In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# =============================================================================
# STAGE 26 — FRESH KAGGLE BOOTSTRAP
# Deployment / Resource Profiling
#
# PURPOSE
# -------
# 1. Verify Kaggle runtime.
# 2. Detect CPU / RAM / GPU environment.
# 3. Recover GitHub token safely from Kaggle Secrets.
# 4. Clone the canonical repository.
# 5. Record exact repository HEAD.
# 6. Inspect attached Kaggle inputs.
# 7. Create Stage 26 local workspace.
#
# IMPORTANT
# ---------
# This cell performs NO model timing and NO scientific measurement.
# The Stage 26 measurement protocol will be frozen separately BEFORE
# benchmark results are observed.
# =============================================================================

from __future__ import annotations

import os
import sys
import json
import shutil
import socket
import platform
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# CONFIG
# =============================================================================

STAGE = 26

REPO_OWNER = "themubasshir"
REPO_NAME = "ids2018-validation-safe-ablation"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")

REPO_DIR = WORK_ROOT / REPO_NAME

STAGE26_ROOT = WORK_ROOT / "stage26_deployment_profiling"

STAGE26_DIRS = {
    "root": STAGE26_ROOT,
    "protocol": STAGE26_ROOT / "protocol",
    "runtime": STAGE26_ROOT / "runtime",
    "raw_timings": STAGE26_ROOT / "raw_timings",
    "cold_start": STAGE26_ROOT / "cold_start",
    "steady_state": STAGE26_ROOT / "steady_state",
    "memory": STAGE26_ROOT / "memory",
    "package_size": STAGE26_ROOT / "package_size",
    "extraction": STAGE26_ROOT / "extraction",
    "pipeline": STAGE26_ROOT / "pipeline",
    "pareto": STAGE26_ROOT / "pareto",
    "figures": STAGE26_ROOT / "figures",
    "tables": STAGE26_ROOT / "tables",
    "logs": STAGE26_ROOT / "logs",
    "workers": STAGE26_ROOT / "workers",
}


# =============================================================================
# HELPERS
# =============================================================================

def banner(text: str):
    print("\n" + "=" * 88)
    print(text)
    print("=" * 88)


def run(
    cmd,
    *,
    cwd=None,
    env=None,
    check=True,
    capture=True,
):
    """Run subprocess without shell expansion."""
    result = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        check=False,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
    )

    if check and result.returncode != 0:
        output = result.stdout if capture else ""
        raise RuntimeError(
            f"Command failed ({result.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"{output}"
        )

    return result


def command_exists(name: str) -> bool:
    return shutil.which(name) is not None


def safe_version(module_name: str):
    try:
        module = __import__(module_name)
        return getattr(module, "__version__", "installed")
    except Exception:
        return None


# =============================================================================
# 1. BASIC RUNTIME
# =============================================================================

banner("STAGE26-BOOTSTRAP :: RUNTIME")

runtime = {
    "stage": STAGE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "hostname": socket.gethostname(),
    "python": sys.version.replace("\n", " "),
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "working_directory": os.getcwd(),
    "kaggle_working_exists": WORK_ROOT.exists(),
    "kaggle_input_exists": INPUT_ROOT.exists(),
}

for k, v in runtime.items():
    print(f"{k:26s}: {v}")


# =============================================================================
# 2. CPU / RAM
# =============================================================================

banner("STAGE26-BOOTSTRAP :: CPU / MEMORY")

try:
    import psutil

    vm = psutil.virtual_memory()

    cpu_info = {
        "physical_cores": psutil.cpu_count(logical=False),
        "logical_cores": psutil.cpu_count(logical=True),
        "cpu_affinity": (
            psutil.Process().cpu_affinity()
            if hasattr(psutil.Process(), "cpu_affinity")
            else None
        ),
        "ram_total_gib": round(vm.total / 1024**3, 3),
        "ram_available_gib": round(vm.available / 1024**3, 3),
    }

except Exception as exc:
    cpu_info = {
        "error": repr(exc),
        "logical_cores_fallback": os.cpu_count(),
    }

for k, v in cpu_info.items():
    print(f"{k:26s}: {v}")


# =============================================================================
# 3. GPU DETECTION
# =============================================================================

banner("STAGE26-BOOTSTRAP :: GPU")

gpu_info = {
    "nvidia_smi_available": command_exists("nvidia-smi"),
    "gpu_present": False,
    "devices": [],
}

if command_exists("nvidia-smi"):
    query = run(
        [
            "nvidia-smi",
            "--query-gpu="
            "index,name,uuid,driver_version,memory.total,"
            "temperature.gpu,utilization.gpu",
            "--format=csv,noheader,nounits",
        ],
        check=False,
    )

    if query.returncode == 0 and query.stdout:
        print(query.stdout.strip())

        for line in query.stdout.strip().splitlines():
            parts = [x.strip() for x in line.split(",")]

            if len(parts) >= 7:
                gpu_info["devices"].append(
                    {
                        "index": parts[0],
                        "name": parts[1],
                        "uuid": parts[2],
                        "driver_version": parts[3],
                        "memory_total_mib": parts[4],
                        "temperature_c": parts[5],
                        "utilization_percent": parts[6],
                    }
                )

        gpu_info["gpu_present"] = len(gpu_info["devices"]) > 0
    else:
        print("[WARN] nvidia-smi exists but GPU query failed.")
else:
    print("[INFO] NVIDIA GPU not detected through nvidia-smi.")


# PyTorch CUDA check
try:
    import torch

    torch_info = {
        "version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "cudnn_version": (
            torch.backends.cudnn.version()
            if torch.backends.cudnn.is_available()
            else None
        ),
        "device_count": torch.cuda.device_count(),
        "devices": [
            torch.cuda.get_device_name(i)
            for i in range(torch.cuda.device_count())
        ],
    }

except Exception as exc:
    torch_info = {
        "available": False,
        "error": repr(exc),
    }

print("\nPyTorch CUDA:")
for k, v in torch_info.items():
    print(f"  {k:22s}: {v}")


# =============================================================================
# 4. IMPORTANT ML PACKAGE VERSIONS
# =============================================================================

banner("STAGE26-BOOTSTRAP :: ML STACK")

packages = [
    "numpy",
    "pandas",
    "sklearn",
    "scipy",
    "xgboost",
    "lightgbm",
    "catboost",
    "torch",
    "tensorflow",
    "psutil",
    "joblib",
    "matplotlib",
]

package_versions = {}

for pkg in packages:
    version = safe_version(pkg)
    package_versions[pkg] = version
    print(f"{pkg:18s}: {version or 'NOT INSTALLED'}")


# =============================================================================
# 5. KAGGLE SECRET DISCOVERY
# =============================================================================

banner("STAGE26-BOOTSTRAP :: GITHUB SECRET")

github_token = None
github_secret_label = None

SECRET_CANDIDATES = [
    "GITHUB_TOKEN",
    "github_token",
    "GH_TOKEN",
    "gh_token",
    "GITHUB_PAT",
    "github_pat",
    "GH_PAT",
    "gh_pat",
]

try:
    from kaggle_secrets import UserSecretsClient

    secret_client = UserSecretsClient()

    for label in SECRET_CANDIDATES:
        try:
            value = secret_client.get_secret(label)

            if value and value.strip():
                github_token = value.strip()
                github_secret_label = label
                print(
                    f"[FOUND] {label} "
                    f"({len(github_token)} characters)"
                )
                break

        except Exception:
            pass

except Exception as exc:
    print(f"[WARN] Kaggle Secrets API unavailable: {exc!r}")


if not github_token:
    raise RuntimeError(
        "\nGitHub token not found or not accessible to this notebook.\n\n"
        "Expected one of these Kaggle Secret labels:\n"
        + "\n".join(f"  - {x}" for x in SECRET_CANDIDATES)
        + "\n\nMake sure the secret is enabled for this notebook."
    )

print(f"\nUsable GitHub secret: {github_secret_label}")


# =============================================================================
# 6. SAFE GIT AUTHENTICATION
# =============================================================================
#
# Do NOT embed the PAT in the repository URL.
# Use GIT_ASKPASS so the token is not printed into notebook output.
# =============================================================================

ASKPASS = WORK_ROOT / ".stage26_git_askpass.sh"

ASKPASS.write_text(
    """#!/bin/sh
case "$1" in
    *Username*) echo "x-access-token" ;;
    *Password*) printf '%s\\n' "$GITHUB_TOKEN" ;;
    *) echo "" ;;
esac
""",
    encoding="utf-8",
)

ASKPASS.chmod(0o700)

git_env = os.environ.copy()
git_env["GITHUB_TOKEN"] = github_token
git_env["GIT_ASKPASS"] = str(ASKPASS)
git_env["GIT_TERMINAL_PROMPT"] = "0"


# =============================================================================
# 7. CLONE / VERIFY REPOSITORY
# =============================================================================

banner("STAGE26-BOOTSTRAP :: GITHUB REPOSITORY")

if not (REPO_DIR / ".git").exists():

    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git repository.\n"
            "Refusing to delete it automatically."
        )

    print(f"Cloning: {REPO_URL}")

    result = run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        env=git_env,
    )

    print(result.stdout.strip())

else:
    print(f"Existing repository detected: {REPO_DIR}")
    print("Fetching origin/main without resetting local files...")

    result = run(
        ["git", "fetch", "origin", "main"],
        cwd=REPO_DIR,
        env=git_env,
    )

    if result.stdout:
        print(result.stdout.strip())


# Remove credential helper immediately after authentication operation
try:
    ASKPASS.unlink()
except FileNotFoundError:
    pass

# Remove token from process environment copy
git_env.pop("GITHUB_TOKEN", None)
github_token = None


# =============================================================================
# 8. REPOSITORY STATE
# =============================================================================

head_sha = run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
).stdout.strip()

branch = run(
    ["git", "branch", "--show-current"],
    cwd=REPO_DIR,
).stdout.strip()

origin_url = run(
    ["git", "remote", "get-url", "origin"],
    cwd=REPO_DIR,
).stdout.strip()

status = run(
    ["git", "status", "--porcelain"],
    cwd=REPO_DIR,
).stdout.strip()

remote_main_sha = run(
    ["git", "rev-parse", "origin/main"],
    cwd=REPO_DIR,
).stdout.strip()

print(f"Repository : {REPO_DIR}")
print(f"Branch     : {branch}")
print(f"HEAD       : {head_sha}")
print(f"origin/main: {remote_main_sha}")
print(f"Origin     : {origin_url}")
print(f"Clean      : {status == ''}")

if status:
    print("\nGit status:")
    print(status)

if head_sha != remote_main_sha:
    print(
        "\n[NOTICE] Local HEAD differs from origin/main."
        "\nNo reset was performed automatically."
    )


# =============================================================================
# 9. CREATE LOCAL STAGE 26 WORKSPACE
# =============================================================================

banner("STAGE26-BOOTSTRAP :: WORKSPACE")

for name, path in STAGE26_DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"{name:16s}: {path}")


# =============================================================================
# 10. INSPECT ATTACHED KAGGLE INPUTS
# =============================================================================

banner("STAGE26-BOOTSTRAP :: ATTACHED KAGGLE INPUTS")

input_inventory = []

if INPUT_ROOT.exists():

    dataset_dirs = sorted(
        p for p in INPUT_ROOT.iterdir()
        if p.is_dir()
    )

    if not dataset_dirs:
        print("[WARN] No Kaggle input datasets detected.")

    for dataset_dir in dataset_dirs:

        files = [
            p for p in dataset_dir.rglob("*")
            if p.is_file()
        ]

        total_bytes = sum(
            p.stat().st_size
            for p in files
        )

        record = {
            "dataset": dataset_dir.name,
            "path": str(dataset_dir),
            "file_count": len(files),
            "total_gib": round(total_bytes / 1024**3, 4),
        }

        input_inventory.append(record)

        print(
            f"{dataset_dir.name:45s} "
            f"files={len(files):6d}  "
            f"size={total_bytes / 1024**3:9.4f} GiB"
        )

else:
    print("[WARN] /kaggle/input does not exist.")


# =============================================================================
# 11. SAVE BOOTSTRAP MANIFEST
# =============================================================================

bootstrap_manifest = {
    "schema": "stage26_bootstrap_v1",
    "stage": 26,

    "scientific_status": (
        "BOOTSTRAP_ONLY__NO_PROFILING_RESULTS_OBSERVED"
    ),

    "timestamp_utc": datetime.now(timezone.utc).isoformat(),

    "repository": {
        "owner": REPO_OWNER,
        "name": REPO_NAME,
        "url": REPO_URL,
        "path": str(REPO_DIR),
        "branch": branch,
        "head_sha": head_sha,
        "origin_main_sha": remote_main_sha,
        "clean": status == "",
    },

    "runtime": runtime,
    "cpu": cpu_info,
    "gpu": gpu_info,
    "torch": torch_info,
    "packages": package_versions,
    "kaggle_inputs": input_inventory,

    "stage26_design_boundary": {
        "protocol_frozen": False,
        "timing_started": False,
        "model_profiling_started": False,
        "feature_extraction_profiling_started": False,
        "pareto_analysis_started": False,
    },
}

manifest_path = (
    STAGE26_DIRS["runtime"]
    / "stage26_bootstrap_manifest.json"
)

manifest_path.write_text(
    json.dumps(
        bootstrap_manifest,
        indent=2,
        sort_keys=True,
        default=str,
    )
    + "\n",
    encoding="utf-8",
)


# =============================================================================
# 12. FINAL AUDIT
# =============================================================================

banner("STAGE26-FRESH-BOOTSTRAP COMPLETE")

print(f"Repository HEAD:")
print(f"  {head_sha}")

print(f"\nRepository clean:")
print(f"  {status == ''}")

print(f"\nGPU present:")
print(f"  {gpu_info['gpu_present']}")

if gpu_info["devices"]:
    for device in gpu_info["devices"]:
        print(
            f"  GPU {device['index']}: "
            f"{device['name']} "
            f"({device['memory_total_mib']} MiB)"
        )

print(f"\nAttached Kaggle datasets:")
print(f"  {len(input_inventory)}")

print(f"\nBootstrap manifest:")
print(f"  {manifest_path}")

print(
    "\nScientific state:\n"
    "  BOOTSTRAP COMPLETE\n"
    "  MEASUREMENT PROTOCOL NOT YET FROZEN\n"
    "  NO LATENCY RESULTS OBSERVED\n"
    "  NO RESOURCE-PROFILING CONCLUSIONS MADE"
)

print(
    "\nNEXT:\n"
    "  STAGE26-0A — inventory the exact frozen models/artifacts available\n"
    "  from GitHub + attached Kaggle datasets BEFORE creating the\n"
    "  measurement_protocol.json lock."
)


STAGE26-BOOTSTRAP :: RUNTIME
stage                     : 26
timestamp_utc             : 2026-08-18T16:34:03.945801+00:00
hostname                  : dcfe416d3de5
python                    : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
python_executable         : /usr/bin/python3
platform                  : Linux-6.12.90+-x86_64-with-glibc2.35
machine                   : x86_64
processor                 : x86_64
working_directory         : /kaggle/working
kaggle_working_exists     : True
kaggle_input_exists       : True

STAGE26-BOOTSTRAP :: CPU / MEMORY
physical_cores            : 2
logical_cores             : 4
cpu_affinity              : [0, 1, 2, 3]
ram_total_gib             : 31.348
ram_available_gib         : 29.673

STAGE26-BOOTSTRAP :: GPU
[INFO] NVIDIA GPU not detected through nvidia-smi.

PyTorch CUDA:
  version               : 2.10.0+cpu
  cuda_available        : False
  cuda_version          : None
  cudnn_version         : None
  device_count          : 0
  devices  

In [3]:
# =============================================================================
# STAGE26-0A — FROZEN ARTIFACT / MODEL INVENTORY
#
# PURPOSE
# -------
# Discover exactly what deployment artifacts are physically available BEFORE
# freezing the Stage 26 measurement protocol.
#
# THIS CELL:
#   - verifies repository HEAD
#   - inventories repository model artifacts
#   - inventories preprocessing/deployment artifacts
#   - inventories candidate model artifacts in /kaggle/input
#   - detects Git-LFS pointer placeholders
#   - computes SHA256 for deployment/model artifacts
#   - searches textual repo material for model-family references
#   - writes machine-readable inventory
#
# THIS CELL DOES NOT:
#   - load any model
#   - perform inference
#   - perform timing
#   - measure memory consumption
#   - select winners
#   - freeze the final Stage 26 model universe
# =============================================================================

from __future__ import annotations

import os
import re
import json
import hashlib
import subprocess
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd


# =============================================================================
# CONFIG
# =============================================================================

EXPECTED_HEAD = "f3957b964e659392c57d1b12881163288f2efedc"

REPO_DIR = Path("/kaggle/working/ids2018-validation-safe-ablation")
INPUT_ROOT = Path("/kaggle/input")

STAGE26_ROOT = Path("/kaggle/working/stage26_deployment_profiling")
RUNTIME_DIR = STAGE26_ROOT / "runtime"
PROTOCOL_DIR = STAGE26_ROOT / "protocol"

RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
PROTOCOL_DIR.mkdir(parents=True, exist_ok=True)


# Extensions strongly associated with serialized ML artifacts.
MODEL_EXTENSIONS = {
    ".pkl",
    ".pickle",
    ".joblib",
    ".model",
    ".cbm",
    ".pt",
    ".pth",
    ".ckpt",
    ".keras",
    ".h5",
    ".hdf5",
    ".onnx",
    ".ubj",
    ".bst",
    ".sav",
    ".tflite",
    ".pb",
}

# Names which may indicate required deployment preprocessing/configuration.
PREPROCESS_TERMS = {
    "scaler",
    "encoder",
    "normalizer",
    "preprocess",
    "preprocessing",
    "feature_columns",
    "feature_names",
    "features",
    "column_order",
    "columns",
    "imputer",
    "threshold",
    "tokenizer",
    "vocab",
    "schema",
    "input_shape",
    "input_geometry",
    "label_encoder",
    "mapping",
}

# Terms whose presence in JSON/TXT may indicate native boosting artifacts.
MODEL_NAME_TERMS = {
    "xgboost",
    "xgb",
    "lightgbm",
    "lgbm",
    "lgb",
    "catboost",
    "cat",
    "cnn",
    "mlp",
    "vit",
    "vision_transformer",
    "transformer",
    "ft_transformer",
    "ft-transformer",
    "fttransformer",
    "tabtransformer",
}

# Reference search terms.
REFERENCE_PATTERNS = {
    "xgboost": re.compile(r"\b(xgboost|xgb)\b", re.I),
    "lightgbm": re.compile(r"\b(lightgbm|lgbm|lgb)\b", re.I),
    "catboost": re.compile(r"\bcatboost\b", re.I),
    "cnn": re.compile(r"\bcnn\b|convolutional", re.I),
    "mlp": re.compile(r"\bmlp\b|multilayer perceptron", re.I),
    "vit": re.compile(r"\bvit\b|vision transformer", re.I),
    "ft_transformer": re.compile(
        r"\bft[-_ ]?transformer\b|feature tokenizer transformer",
        re.I,
    ),
    "transformer": re.compile(r"\btransformer\b", re.I),
}

TEXT_EXTENSIONS = {
    ".md",
    ".txt",
    ".json",
    ".py",
    ".yaml",
    ".yml",
    ".toml",
}

# Avoid scanning giant generated files as text.
MAX_TEXT_SCAN_BYTES = 5 * 1024 * 1024


# =============================================================================
# HELPERS
# =============================================================================

def banner(text: str):
    print("\n" + "=" * 96)
    print(text)
    print("=" * 96)


def git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(result.stdout)

    return result.stdout.strip()


def human_size(n: int) -> str:
    units = ["B", "KiB", "MiB", "GiB", "TiB"]

    value = float(n)

    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:.3f} {unit}"
        value /= 1024

    return f"{n} B"


def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def is_lfs_pointer(path: Path) -> bool:
    """
    Detect a Git-LFS pointer without relying on git-lfs availability.
    """
    try:
        if path.stat().st_size > 4096:
            return False

        raw = path.read_bytes()[:512]

        return (
            b"version https://git-lfs.github.com/spec/v1"
            in raw
        )

    except Exception:
        return False


def relative_source_path(path: Path):
    try:
        return "repository", str(path.relative_to(REPO_DIR))
    except ValueError:
        pass

    try:
        return "kaggle_input", str(path.relative_to(INPUT_ROOT))
    except ValueError:
        pass

    return "other", str(path)


def classify_family(path: Path) -> str:
    s = str(path).lower()

    if "catboost" in s:
        return "catboost"

    if "lightgbm" in s or "lgbm" in s:
        return "lightgbm"

    if "xgboost" in s or re.search(r"(^|[/_.-])xgb([/_.-]|$)", s):
        return "xgboost"

    if "ft_transformer" in s or "ft-transformer" in s or "fttransformer" in s:
        return "ft_transformer"

    if "vision_transformer" in s or re.search(r"(^|[/_.-])vit([/_.-]|$)", s):
        return "vit"

    if "transformer" in s:
        return "transformer"

    if re.search(r"(^|[/_.-])cnn([/_.-]|$)", s):
        return "cnn"

    if re.search(r"(^|[/_.-])mlp([/_.-]|$)", s):
        return "mlp"

    return "unknown"


def classify_artifact(path: Path) -> str:
    lower_name = path.name.lower()
    lower_path = str(path).lower()
    suffix = path.suffix.lower()

    if suffix in MODEL_EXTENSIONS:
        return "model"

    # All regular files beneath repository models/ are relevant artifacts.
    try:
        rel = path.relative_to(REPO_DIR)

        if rel.parts and rel.parts[0] == "models":
            return "model_support"
    except ValueError:
        pass

    if any(term in lower_name for term in PREPROCESS_TERMS):
        return "preprocessing"

    # Native tree-model JSON/TXT artifacts may not have a special extension.
    if suffix in {".json", ".txt"}:
        if any(term in lower_path for term in MODEL_NAME_TERMS):
            return "model_support"

    return "other"


def should_hash(path: Path, artifact_type: str) -> bool:
    """
    Hash deployment/model-related artifacts, but do not hash arbitrary
    multi-GB dataset files merely for this discovery step.
    """
    return artifact_type in {
        "model",
        "model_support",
        "preprocessing",
    }


# =============================================================================
# 1. REPOSITORY INTEGRITY
# =============================================================================

banner("STAGE26-0A :: REPOSITORY INTEGRITY")

actual_head = git("rev-parse", "HEAD")
status = git("status", "--porcelain")

print("Expected HEAD :", EXPECTED_HEAD)
print("Actual HEAD   :", actual_head)
print("Clean repo    :", status == "")

if actual_head != EXPECTED_HEAD:
    raise RuntimeError(
        "Repository HEAD changed after Stage26 bootstrap.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {actual_head}"
    )

if status:
    raise RuntimeError(
        "Repository is not clean before Stage26 artifact inventory:\n"
        + status
    )


# =============================================================================
# 2. REPOSITORY MODEL TREE
# =============================================================================

banner("STAGE26-0A :: REPOSITORY MODEL TREE")

models_root = REPO_DIR / "models"

if not models_root.exists():
    raise RuntimeError(f"Missing repository models directory: {models_root}")

top_model_dirs = sorted(
    p.name
    for p in models_root.iterdir()
    if p.is_dir()
)

print("Top-level model-family directories:")

for name in top_model_dirs:
    print("  -", name)

print("\nTotal:", len(top_model_dirs))


# =============================================================================
# 3. COLLECT ARTIFACT CANDIDATES
# =============================================================================

banner("STAGE26-0A :: COLLECTING ARTIFACT CANDIDATES")

records = []


def add_artifact(path: Path, force=False):
    if not path.is_file():
        return

    artifact_type = classify_artifact(path)

    if not force and artifact_type == "other":
        return

    source, relative_path = relative_source_path(path)

    size = path.stat().st_size
    lfs_pointer = is_lfs_pointer(path)

    digest = None

    if should_hash(path, artifact_type):
        digest = sha256_file(path)

    records.append(
        {
            "source": source,
            "relative_path": relative_path,
            "absolute_path": str(path),
            "filename": path.name,
            "suffix": path.suffix.lower(),
            "family_guess": classify_family(path),
            "artifact_type": artifact_type,
            "size_bytes": size,
            "size_human": human_size(size),
            "sha256": digest,
            "git_lfs_pointer": lfs_pointer,
        }
    )


# -----------------------------------------------------------------------------
# 3A. Everything below repo/models
# -----------------------------------------------------------------------------

for path in sorted(models_root.rglob("*")):
    if path.is_file():
        add_artifact(path, force=True)


# -----------------------------------------------------------------------------
# 3B. Preprocessing/deployment candidates elsewhere in repository
# -----------------------------------------------------------------------------

for root_name in ["metadata", "results", "scripts"]:
    root = REPO_DIR / root_name

    if not root.exists():
        continue

    for path in sorted(root.rglob("*")):

        if not path.is_file():
            continue

        lower_name = path.name.lower()
        lower_path = str(path).lower()

        if (
            path.suffix.lower() in MODEL_EXTENSIONS
            or any(term in lower_name for term in PREPROCESS_TERMS)
            or (
                path.suffix.lower() in {".json", ".txt"}
                and any(term in lower_path for term in MODEL_NAME_TERMS)
            )
        ):
            add_artifact(path, force=True)


# -----------------------------------------------------------------------------
# 3C. Candidate deployment artifacts from attached Kaggle datasets
# -----------------------------------------------------------------------------

if INPUT_ROOT.exists():

    for path in sorted(INPUT_ROOT.rglob("*")):

        if not path.is_file():
            continue

        suffix = path.suffix.lower()
        lower_name = path.name.lower()
        lower_path = str(path).lower()

        candidate = (
            suffix in MODEL_EXTENSIONS
            or any(term in lower_name for term in PREPROCESS_TERMS)
            or (
                suffix in {".json", ".txt"}
                and any(term in lower_path for term in MODEL_NAME_TERMS)
            )
        )

        if candidate:
            add_artifact(path, force=True)


inventory_df = pd.DataFrame(records)

if inventory_df.empty:
    raise RuntimeError("No deployment/model artifacts were discovered.")


# =============================================================================
# 4. DUPLICATE SHA ANALYSIS
# =============================================================================

banner("STAGE26-0A :: ARTIFACT SUMMARY")

print("Artifacts discovered :", len(inventory_df))
print()

print("By source:")
print(inventory_df["source"].value_counts().to_string())

print("\nBy artifact type:")
print(inventory_df["artifact_type"].value_counts().to_string())

print("\nBy family guess:")
print(inventory_df["family_guess"].value_counts().to_string())

print("\nGit-LFS pointer files:")
print(int(inventory_df["git_lfs_pointer"].sum()))


hashed = inventory_df[
    inventory_df["sha256"].notna()
].copy()

sha_counts = hashed["sha256"].value_counts()

duplicate_hashes = set(
    sha_counts[sha_counts > 1].index
)

inventory_df["duplicate_sha256"] = (
    inventory_df["sha256"]
    .apply(lambda x: x in duplicate_hashes if pd.notna(x) else False)
)


# =============================================================================
# 5. SHOW MODEL/DEPLOYMENT ARTIFACTS
# =============================================================================

banner("STAGE26-0A :: DISCOVERED DEPLOYMENT ARTIFACTS")

display_cols = [
    "source",
    "family_guess",
    "artifact_type",
    "relative_path",
    "size_human",
    "git_lfs_pointer",
]

view = (
    inventory_df[display_cols]
    .sort_values(
        ["family_guess", "source", "relative_path"]
    )
    .reset_index(drop=True)
)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

print(view.to_string(index=False))


# =============================================================================
# 6. FULL KAGGLE INPUT FILE INVENTORY
# =============================================================================
#
# Record all input files by path and size, but DO NOT SHA256 every multi-GB data
# file during this model-discovery stage.
# =============================================================================

banner("STAGE26-0A :: KAGGLE INPUT INVENTORY")

input_records = []

if INPUT_ROOT.exists():

    for path in sorted(INPUT_ROOT.rglob("*")):

        if not path.is_file():
            continue

        stat = path.stat()

        input_records.append(
            {
                "relative_path": str(path.relative_to(INPUT_ROOT)),
                "size_bytes": stat.st_size,
                "size_human": human_size(stat.st_size),
                "suffix": path.suffix.lower(),
                "family_guess": classify_family(path),
                "artifact_type": classify_artifact(path),
            }
        )

input_df = pd.DataFrame(input_records)

print("Input files:", len(input_df))

if not input_df.empty:
    print(
        input_df[
            [
                "relative_path",
                "size_human",
                "suffix",
                "family_guess",
                "artifact_type",
            ]
        ].to_string(index=False)
    )


# =============================================================================
# 7. SEARCH REPOSITORY TEXT FOR ARCHITECTURE REFERENCES
# =============================================================================

banner("STAGE26-0A :: ARCHITECTURE REFERENCE SEARCH")

reference_hits = []

SEARCH_ROOTS = [
    REPO_DIR / "docs",
    REPO_DIR / "metadata",
    REPO_DIR / "results",
    REPO_DIR / "scripts",
    REPO_DIR / "notebooks",
]

for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() not in TEXT_EXTENSIONS:
            continue

        try:
            if path.stat().st_size > MAX_TEXT_SCAN_BYTES:
                continue

            text = path.read_text(
                encoding="utf-8",
                errors="ignore",
            )

        except Exception:
            continue

        lines = text.splitlines()

        for line_no, line in enumerate(lines, start=1):

            for family, pattern in REFERENCE_PATTERNS.items():

                if pattern.search(line):

                    reference_hits.append(
                        {
                            "family": family,
                            "path": str(path.relative_to(REPO_DIR)),
                            "line": line_no,
                            "text": line.strip()[:500],
                        }
                    )


reference_df = pd.DataFrame(reference_hits)

if reference_df.empty:
    print("No architecture references found.")

else:
    counts = (
        reference_df
        .groupby("family")
        .size()
        .sort_values(ascending=False)
    )

    print("Reference-hit counts:")
    print(counts.to_string())

    print("\nFiles mentioning each family:")

    for family in sorted(reference_df["family"].unique()):

        files = sorted(
            reference_df.loc[
                reference_df["family"] == family,
                "path",
            ].unique()
        )

        print(f"\n[{family}]")

        for file in files[:40]:
            print("  ", file)

        if len(files) > 40:
            print(
                f"   ... +{len(files) - 40} additional files"
            )


# =============================================================================
# 8. SPECIAL CHECK — STAGE26 PLANNED FAMILIES
# =============================================================================

banner("STAGE26-0A :: PLANNED-FAMILY AVAILABILITY CHECK")

PLANNED_FAMILIES = [
    "xgboost",
    "lightgbm",
    "catboost",
    "cnn",
    "vit",
    "ft_transformer",
]

availability_rows = []

for family in PLANNED_FAMILIES:

    physical_artifacts = inventory_df[
        inventory_df["family_guess"] == family
    ]

    if reference_df.empty:
        refs = pd.DataFrame()
    else:
        refs = reference_df[
            reference_df["family"] == family
        ]

    availability_rows.append(
        {
            "family": family,
            "physical_artifact_count": len(physical_artifacts),
            "reference_hit_count": len(refs),
            "has_physical_candidate": len(physical_artifacts) > 0,
            "has_repo_reference": len(refs) > 0,
        }
    )

availability_df = pd.DataFrame(availability_rows)

print(availability_df.to_string(index=False))


# =============================================================================
# 9. EXTRA DISCOVERED FAMILIES
# =============================================================================

banner("STAGE26-0A :: EXTRA DISCOVERED MODEL FAMILIES")

planned_set = set(PLANNED_FAMILIES)

extra_families = sorted(
    family
    for family in inventory_df["family_guess"].dropna().unique()
    if family not in planned_set
    and family != "unknown"
)

if extra_families:
    for family in extra_families:
        count = int(
            (inventory_df["family_guess"] == family).sum()
        )
        print(f"{family}: {count} candidate artifacts")
else:
    print("None.")


# =============================================================================
# 10. SAVE MACHINE-READABLE OUTPUTS
# =============================================================================

banner("STAGE26-0A :: SAVING INVENTORY")

inventory_csv = RUNTIME_DIR / "stage26_artifact_inventory.csv"
inventory_json = RUNTIME_DIR / "stage26_artifact_inventory.json"

input_csv = RUNTIME_DIR / "stage26_kaggle_input_inventory.csv"

references_csv = RUNTIME_DIR / "stage26_model_reference_hits.csv"
availability_csv = RUNTIME_DIR / "stage26_planned_family_availability.csv"


inventory_df.to_csv(
    inventory_csv,
    index=False,
)

input_df.to_csv(
    input_csv,
    index=False,
)

reference_df.to_csv(
    references_csv,
    index=False,
)

availability_df.to_csv(
    availability_csv,
    index=False,
)


inventory_payload = {
    "schema": "stage26_artifact_inventory_v1",
    "stage": 26,
    "repository_head": actual_head,

    "scientific_status": (
        "ARTIFACT_DISCOVERY_ONLY__NO_MODELS_LOADED__NO_TIMING"
    ),

    "top_level_model_directories": top_model_dirs,

    "planned_families": PLANNED_FAMILIES,

    "extra_discovered_families": extra_families,

    "artifacts": inventory_df.to_dict(
        orient="records"
    ),

    "planned_family_availability": availability_df.to_dict(
        orient="records"
    ),

    "design_boundary": {
        "protocol_frozen": False,
        "models_loaded": False,
        "timing_started": False,
        "memory_profiling_started": False,
        "gpu_used": False,
    },
}

inventory_json.write_text(
    json.dumps(
        inventory_payload,
        indent=2,
        sort_keys=True,
        default=str,
    )
    + "\n",
    encoding="utf-8",
)


print("Artifact CSV:")
print(" ", inventory_csv)

print("\nArtifact JSON:")
print(" ", inventory_json)

print("\nKaggle input inventory:")
print(" ", input_csv)

print("\nReference hits:")
print(" ", references_csv)

print("\nFamily availability:")
print(" ", availability_csv)


# =============================================================================
# 11. FINAL AUDIT
# =============================================================================

banner("STAGE26-0A COMPLETE")

print("Repository HEAD:")
print(" ", actual_head)

print("\nTop-level model directories:")
for name in top_model_dirs:
    print(" ", name)

print("\nArtifact candidates:")
print(" ", len(inventory_df))

print("\nPlanned-family availability:")
print(availability_df.to_string(index=False))

print(
    "\nScientific state:\n"
    "  ARTIFACT INVENTORY COMPLETE\n"
    "  NO MODEL LOADED\n"
    "  NO INFERENCE EXECUTED\n"
    "  NO LATENCY MEASURED\n"
    "  NO MEMORY PROFILE MEASURED\n"
    "  GPU NOT USED\n"
    "  MEASUREMENT PROTOCOL NOT YET FROZEN"
)

print(
    "\nNEXT:\n"
    "  STAGE26-0B — resolve which exact artifact belongs to each candidate\n"
    "  architecture, determine required preprocessing/input representation,\n"
    "  and classify models into scientifically comparable evaluation groups.\n"
    "\n"
    "  DO NOT freeze measurement_protocol.json until 0B is resolved."
)


STAGE26-0A :: REPOSITORY INTEGRITY
Expected HEAD : f9307d5f7a4c8fd53886d66b3e5f8e74a65d87c6
Actual HEAD   : f9307d5f7a4c8fd53886d66b3e5f8e74a65d87c6
Clean repo    : True

STAGE26-0A :: REPOSITORY MODEL TREE
Top-level model-family directories:
  - catboost
  - cnn
  - lightgbm
  - mlp
  - xgboost

Total: 5

STAGE26-0A :: COLLECTING ARTIFACT CANDIDATES

STAGE26-0A :: ARTIFACT SUMMARY
Artifacts discovered : 496

By source:
source
repository      488
kaggle_input      8

By artifact type:
artifact_type
model_support    288
preprocessing    125
model             83

By family guess:
family_guess
transformer    147
unknown        128
xgboost         93
lightgbm        80
vit             21
catboost        20
cnn              4
mlp              3

Git-LFS pointer files:
0

STAGE26-0A :: DISCOVERED DEPLOYMENT ARTIFACTS
      source family_guess artifact_type                                                                                                                                         

In [4]:
# =============================================================================
# STAGE26-0B — EXACT ARTIFACT + COMPARABILITY RESOLUTION
#
# PURPOSE
# -------
# Resolve the exact frozen deployment artifacts that Stage 26 is permitted
# to profile, together with their preprocessing dependencies and scientifically
# valid comparison populations.
#
# IMPORTANT
# ---------
# This cell DOES NOT:
#   - deserialize/load models
#   - execute inference
#   - benchmark latency
#   - measure RAM/GPU memory
#   - open holdout data
#   - recompute predictive metrics
#
# Stage26 measurement_protocol.json remains UNFROZEN after this cell.
# =============================================================================

from __future__ import annotations

import json
import hashlib
import subprocess
from pathlib import Path

import pandas as pd


# =============================================================================
# CONFIG
# =============================================================================

EXPECTED_HEAD = "f3957b964e659392c57d1b12881163288f2efedc"

REPO_DIR = Path("/kaggle/working/ids2018-validation-safe-ablation")

STAGE26_ROOT = Path("/kaggle/working/stage26_deployment_profiling")
RUNTIME_DIR = STAGE26_ROOT / "runtime"
PROTOCOL_DIR = STAGE26_ROOT / "protocol"

RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
PROTOCOL_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# HELPERS
# =============================================================================

def banner(text: str):
    print("\n" + "=" * 100)
    print(text)
    print("=" * 100)


def git(*args):
    p = subprocess.run(
        ["git", *args],
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(p.stdout)

    return p.stdout.strip()


def sha256_file(path: Path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def relative(path: Path):
    return str(path.relative_to(REPO_DIR))


def artifact_record(
    *,
    artifact_id,
    family,
    role,
    path,
    comparison_group,
    representation,
    expected_sha256=None,
    expected_size_bytes=None,
    notes=None,
):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Required Stage26 artifact missing:\n{path}"
        )

    if not path.is_file():
        raise RuntimeError(
            f"Expected file but found something else:\n{path}"
        )

    actual_size = path.stat().st_size
    actual_sha = sha256_file(path)

    hash_match = (
        None
        if expected_sha256 is None
        else actual_sha == expected_sha256
    )

    size_match = (
        None
        if expected_size_bytes is None
        else actual_size == expected_size_bytes
    )

    return {
        "artifact_id": artifact_id,
        "family": family,
        "role": role,
        "comparison_group": comparison_group,
        "representation": representation,
        "repo_relative_path": relative(path),
        "absolute_path": str(path),
        "size_bytes": actual_size,
        "sha256": actual_sha,
        "expected_sha256": expected_sha256,
        "sha256_match": hash_match,
        "expected_size_bytes": expected_size_bytes,
        "size_match": size_match,
        "notes": notes,
    }


# =============================================================================
# 1. REPOSITORY GATE
# =============================================================================

banner("STAGE26-0B :: REPOSITORY GATE")

HEAD = git("rev-parse", "HEAD")
STATUS = git("status", "--porcelain")

print("Expected HEAD :", EXPECTED_HEAD)
print("Actual HEAD   :", HEAD)
print("Clean repo    :", STATUS == "")

if HEAD != EXPECTED_HEAD:
    raise RuntimeError(
        "Repository HEAD changed after Stage26-0A."
    )

if STATUS:
    raise RuntimeError(
        "Repository became dirty before Stage26-0B:\n"
        + STATUS
    )


# =============================================================================
# 2. LOAD AUTHORITATIVE UPSTREAM LOCKS
# =============================================================================

banner("STAGE26-0B :: AUTHORITATIVE UPSTREAM LOCKS")

STAGE15_FROZEN_ARCH_PATH = (
    REPO_DIR
    / "results/stage15_transformer_checkpoint/"
      "stage15_4c_frozen_architecture.json"
)

STAGE15_PREHOLDOUT_PATH = (
    REPO_DIR
    / "results/stage15_transformer_checkpoint/"
      "stage15_5b_preholdout_decision_lock.json"
)

STAGE15_HOLDOUT_PATH = (
    REPO_DIR
    / "results/stage15_transformer_checkpoint/"
      "stage15_6a_holdout_evaluation_result.json"
)

STAGE16_COMPARABILITY_PATH = (
    REPO_DIR
    / "results/stage16_classical_benchmark_checkpoint/"
      "stage16_0_classical_comparability_audit.json"
)

STAGE16_TUNED_MANIFEST_PATH = (
    REPO_DIR
    / "results/stage16_classical_benchmark_checkpoint/"
      "stage16_3b_tuned_model_manifest.json"
)

STAGE16_FINAL_PATH = (
    REPO_DIR
    / "results/stage16_classical_benchmark_checkpoint/"
      "stage16_6c_final_classical_holdout_lock.json"
)

STAGE21_PROTOCOL_PATH = (
    REPO_DIR
    / "results/stage21_architecture/"
      "stage21_0_cnn_vit_followup_protocol_lock.json"
)

STAGE21_COMPARISON_PATH = (
    REPO_DIR
    / "results/stage21_architecture/"
      "stage21_5_cnn_vit_descriptive_comparison.json"
)


AUTHORITATIVE_PATHS = [
    STAGE15_FROZEN_ARCH_PATH,
    STAGE15_PREHOLDOUT_PATH,
    STAGE15_HOLDOUT_PATH,
    STAGE16_COMPARABILITY_PATH,
    STAGE16_TUNED_MANIFEST_PATH,
    STAGE16_FINAL_PATH,
    STAGE21_PROTOCOL_PATH,
    STAGE21_COMPARISON_PATH,
]

for path in AUTHORITATIVE_PATHS:
    if not path.exists():
        raise FileNotFoundError(path)

    print(
        f"[OK] {relative(path)}"
        f"  sha256={sha256_file(path)[:16]}..."
    )


stage15_arch = load_json(STAGE15_FROZEN_ARCH_PATH)
stage15_lock = load_json(STAGE15_PREHOLDOUT_PATH)
stage15_holdout = load_json(STAGE15_HOLDOUT_PATH)

stage16_cmp = load_json(STAGE16_COMPARABILITY_PATH)
stage16_manifest = load_json(STAGE16_TUNED_MANIFEST_PATH)
stage16_final = load_json(STAGE16_FINAL_PATH)

stage21_protocol = load_json(STAGE21_PROTOCOL_PATH)
stage21_cmp = load_json(STAGE21_COMPARISON_PATH)


# =============================================================================
# 3. STAGE 15 FT-TRANSFORMER RESOLUTION
# =============================================================================

banner("STAGE26-0B :: FT-TRANSFORMER RESOLUTION")

if stage15_arch["candidate_id"] != "FT_BALANCED":
    raise RuntimeError(
        "Unexpected frozen Stage15 architecture."
    )

if stage15_arch["input_predictor_count"] != 70:
    raise RuntimeError(
        "Unexpected Stage15 predictor count."
    )

primary_ensemble = stage15_holdout["primary_ensemble_result"]

if primary_ensemble["model"] != "five_checkpoint_soft_voting_ensemble":
    raise RuntimeError(
        "Unexpected Stage15 final predictor policy."
    )

if primary_ensemble["checkpoint_count"] != 5:
    raise RuntimeError(
        "Stage15 final ensemble does not contain five checkpoints."
    )

print("Candidate ID             :", stage15_arch["candidate_id"])
print("Input predictors         :", stage15_arch["input_predictor_count"])
print("Input representation     :", stage15_arch["input_representation"])
print("Final predictor policy   :", primary_ensemble["model"])
print("Checkpoint count         :", primary_ensemble["checkpoint_count"])
print("Checkpoint seeds         :", primary_ensemble["checkpoint_seeds"])
print("Ensemble method          :", primary_ensemble["ensemble_method"])


# Map old Kaggle working paths in the Stage15 lock to committed repo paths.
FT_PATH_MAP = {
    7: (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_4b_models/FT_BALANCED_seed_7_best_extended.pt"
    ),
    29: (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_4a_models/FT_BALANCED_seed_29_best.pt"
    ),
    101: (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_4a_models/FT_BALANCED_seed_101_best.pt"
    ),
    313: (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_4c_models/FT_BALANCED_seed_313_best.pt"
    ),
    997: (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_4c_models/FT_BALANCED_seed_997_best.pt"
    ),
}

FT_CODE_PATH = (
    REPO_DIR
    / "results/stage15_transformer_checkpoint/"
      "ft_transformer_numeric.py"
)

FT_SCALER_PATH = (
    REPO_DIR
    / "results/stage15_transformer_checkpoint/"
      "stage15_2_standard_scaler.joblib"
)

for path in [FT_CODE_PATH, FT_SCALER_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)


ft_records = []

for seed in [7, 29, 101, 313, 997]:

    lock = stage15_lock["checkpoint_locks"][str(seed)]

    record = artifact_record(
        artifact_id=f"FT_BALANCED_seed_{seed}",
        family="ft_transformer",
        role="FROZEN_ENSEMBLE_MEMBER",
        path=FT_PATH_MAP[seed],
        comparison_group="GROUP_A_DUPSAFE70",
        representation="70_FEATURE_SCALED_NUMERIC_TOKENIZER",
        expected_sha256=lock["sha256"],
        expected_size_bytes=lock["size_bytes"],
        notes=(
            "Member of frozen Stage15 five-checkpoint "
            "soft-voting ensemble."
        ),
    )

    if record["sha256_match"] is not True:
        raise RuntimeError(
            f"FT checkpoint SHA mismatch for seed {seed}"
        )

    if record["size_match"] is not True:
        raise RuntimeError(
            f"FT checkpoint size mismatch for seed {seed}"
        )

    ft_records.append(record)

    print(
        f"[VERIFIED] seed={seed:<3} "
        f"{record['size_bytes']:>8} bytes  "
        f"{record['sha256'][:16]}..."
    )


# Resource-only single-checkpoint representative.
#
# This is NOT model selection.
# We freeze seed 7 solely because it is the numerically lowest confirmation
# seed in the already-frozen set. Predictive performance is not consulted.
FT_RESOURCE_REPRESENTATIVE_SEED = min(
    primary_ensemble["checkpoint_seeds"]
)

print(
    "\nResource-only representative checkpoint seed:",
    FT_RESOURCE_REPRESENTATIVE_SEED,
)

print(
    "Selection rule: numerically lowest member of frozen "
    "confirmation seed set; performance-independent."
)


# =============================================================================
# 4. STAGE 16 CLASSICAL ARTIFACT RESOLUTION
# =============================================================================

banner("STAGE26-0B :: STAGE16 CLASSICAL ARTIFACT RESOLUTION")

if stage16_cmp["audit_decision"] != (
    "RETRAIN_CLASSICAL_MODELS_ON_EXACT_DUPLICATE_SAFE_SPLIT"
):
    raise RuntimeError(
        "Unexpected Stage16 comparability policy."
    )

print("Stage16 comparability decision:")
print(" ", stage16_cmp["audit_decision"])

print("\nReason:")
print(" ", stage16_cmp["audit_reason"])


manifest_by_id = {
    item["candidate_id"]: item
    for item in stage16_manifest["models"]
}

PRIMARY_CLASSICAL_IDS = [
    "XGBOOST",
    "LIGHTGBM",
    "CATBOOST",
]

CLASSICAL_REPO_PATHS = {
    "XGBOOST": (
        REPO_DIR
        / "results/stage16_classical_benchmark_checkpoint/"
          "stage16_3_tuned_models/XGBOOST_tuned.joblib"
    ),
    "LIGHTGBM": (
        REPO_DIR
        / "results/stage16_classical_benchmark_checkpoint/"
          "stage16_3_tuned_models/LIGHTGBM_tuned.joblib"
    ),
    "CATBOOST": (
        REPO_DIR
        / "results/stage16_classical_benchmark_checkpoint/"
          "stage16_3_tuned_models/CATBOOST_tuned.joblib"
    ),
}

classical_records = []

for candidate_id in PRIMARY_CLASSICAL_IDS:

    meta = manifest_by_id[candidate_id]

    record = artifact_record(
        artifact_id=f"STAGE16_{candidate_id}_TUNED",
        family=candidate_id.lower(),
        role="PRIMARY_ARCHITECTURE_PROFILE",
        path=CLASSICAL_REPO_PATHS[candidate_id],
        comparison_group="GROUP_A_DUPSAFE70",
        representation=(
            "70_FEATURE_RAW_NUMERIC"
            if meta["input_representation"] == "raw"
            else f"70_FEATURE_{meta['input_representation'].upper()}"
        ),
        expected_sha256=meta["model_sha256"],
        expected_size_bytes=meta["model_size_bytes"],
        notes=(
            f"Stage16 winning configuration "
            f"{meta['winning_configuration_id']}; "
            "fit on duplicate-safe training only; "
            "selected on duplicate-safe validation only."
        ),
    )

    if record["sha256_match"] is not True:
        raise RuntimeError(
            f"{candidate_id} SHA mismatch"
        )

    if record["size_match"] is not True:
        raise RuntimeError(
            f"{candidate_id} size mismatch"
        )

    record["validation_pr_auc"] = meta["validation_pr_auc"]
    record["validation_f1"] = meta["validation_f1"]
    record["selected_threshold"] = meta["selected_threshold"]
    record["winning_configuration_id"] = meta[
        "winning_configuration_id"
    ]
    record["training_backend_recorded"] = meta[
        "winning_parameters"
    ]

    classical_records.append(record)

    print(
        f"[VERIFIED] {candidate_id:10s} "
        f"{record['size_bytes']:>9} bytes  "
        f"PR-AUC={meta['validation_pr_auc']:.9f}  "
        f"{record['sha256'][:16]}..."
    )


# =============================================================================
# 5. FINAL CLASSICAL OPERATIONAL REFERENCE
# =============================================================================

banner("STAGE26-0B :: FINAL CLASSICAL OPERATIONAL REFERENCE")

final_strategy = stage16_final["final_classical_strategy"]

if final_strategy["strategy_id"] != "ENS_LGBM_XGB_EQUAL":
    raise RuntimeError(
        "Unexpected final Stage16 classical strategy."
    )

print("Strategy ID     :", final_strategy["strategy_id"])
print("Status          :", final_strategy["strategy_status"])
print("Threshold       :", final_strategy["operating_threshold"])
print("Members:")

for member in final_strategy["members"]:
    print("  -", member)

print(
    "\nThis is retained as an OPERATIONAL REFERENCE,"
    "\nnot counted as an additional architecture family."
)


# =============================================================================
# 6. STAGE20 CNN + STAGE21 ViT RESOLUTION
# =============================================================================

banner("STAGE26-0B :: PACKET-IMAGE CNN / VIT RESOLUTION")

representation = stage21_protocol[
    "representation_and_supervision"
]["representation"]

if representation != "64x256x1_FROZEN_STAGE20_PACKET_IMAGE":
    raise RuntimeError(
        "Unexpected Stage20/21 packet-image representation."
    )

friday_population = stage21_cmp["population"]

if friday_population["flows"] != 12088:
    raise RuntimeError(
        "Unexpected Friday comparison population."
    )


CNN_STATE = (
    REPO_DIR
    / "results/stage20_1e_training/"
      "stage20_1e2_epoch10_model_state_dict.pt"
)

VIT_STATE = (
    REPO_DIR
    / "results/stage21_architecture/"
      "stage21_2_epoch10_model_state_dict.pt"
)

PACKET_ENCODER = (
    REPO_DIR
    / "scripts/stage20_packet_image_encoder.py"
)

VIT_EXECUTED_TRAIN_SCRIPT = (
    REPO_DIR
    / "results/stage21_architecture/"
      "stage21_2_train_fast_executed.py"
)

for path in [
    CNN_STATE,
    VIT_STATE,
    PACKET_ENCODER,
    VIT_EXECUTED_TRAIN_SCRIPT,
]:
    if not path.exists():
        raise FileNotFoundError(path)


cnn_record = artifact_record(
    artifact_id="STAGE20_MASKED_CNN_V1",
    family="cnn",
    role="PRIMARY_ARCHITECTURE_PROFILE",
    path=CNN_STATE,
    comparison_group="GROUP_B_PACKET_IMAGE",
    representation=representation,
    notes=(
        "Immutable Stage20 CNN comparator reused by Stage21; "
        "no retraining permitted in Stage21."
    ),
)

vit_record = artifact_record(
    artifact_id="STAGE21_MASKED_VIT_V1",
    family="vit",
    role="PRIMARY_ARCHITECTURE_PROFILE",
    path=VIT_STATE,
    comparison_group="GROUP_B_PACKET_IMAGE",
    representation=representation,
    notes=(
        "Single frozen Stage21MaskedViTv1 candidate; "
        "trained for exactly 10 epochs under preregistered protocol."
    ),
)


print("Representation :", representation)

print(
    "Friday corpus  :",
    friday_population["flows"],
    "flows =",
    friday_population["benign"],
    "benign +",
    friday_population["attack"],
    "attack",
)

print("\nCNN:")
print(" ", relative(CNN_STATE))
print(" ", cnn_record["size_bytes"], "bytes")
print(" ", cnn_record["sha256"])

print("\nViT:")
print(" ", relative(VIT_STATE))
print(" ", vit_record["size_bytes"], "bytes")
print(" ", vit_record["sha256"])

print("\nPersisted Friday ranking metrics:")
print(
    "  CNN PR-AUC:",
    stage21_cmp["co_primary_descriptive"]["CNN_PR_AUC"]
)
print(
    "  ViT PR-AUC:",
    stage21_cmp["co_primary_descriptive"]["ViT_PR_AUC"]
)

print(
    "\nClaim boundary:",
    stage21_cmp["role"]
)


# =============================================================================
# 7. PREPROCESSING / ARCHITECTURE DEPENDENCIES
# =============================================================================

banner("STAGE26-0B :: REQUIRED DEPLOYMENT DEPENDENCIES")

dependency_records = []


def dependency(
    artifact_id,
    family,
    path,
    role,
    comparison_group,
):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    rec = {
        "artifact_id": artifact_id,
        "family": family,
        "role": role,
        "comparison_group": comparison_group,
        "repo_relative_path": relative(path),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }

    dependency_records.append(rec)
    return rec


dependency(
    "FT_STANDARD_SCALER",
    "ft_transformer",
    FT_SCALER_PATH,
    "REQUIRED_PREPROCESSING",
    "GROUP_A_DUPSAFE70",
)

dependency(
    "FT_ARCHITECTURE_CODE",
    "ft_transformer",
    FT_CODE_PATH,
    "REQUIRED_EXECUTABLE_ARCHITECTURE",
    "GROUP_A_DUPSAFE70",
)

dependency(
    "PACKET_IMAGE_ENCODER",
    "cnn_vit_shared",
    PACKET_ENCODER,
    "REQUIRED_REPRESENTATION_CODE",
    "GROUP_B_PACKET_IMAGE",
)

dependency(
    "VIT_ARCHITECTURE_EXECUTED_SOURCE",
    "vit",
    VIT_EXECUTED_TRAIN_SCRIPT,
    "REQUIRED_EXECUTABLE_ARCHITECTURE",
    "GROUP_B_PACKET_IMAGE",
)


for rec in dependency_records:
    print(
        f"[{rec['family']}] "
        f"{rec['role']}\n"
        f"  {rec['repo_relative_path']}\n"
        f"  {rec['size_bytes']} bytes\n"
        f"  {rec['sha256'][:20]}...\n"
    )


# =============================================================================
# 8. PRIMARY STAGE26 MODEL UNIVERSE
# =============================================================================

banner("STAGE26-0B :: PRIMARY MODEL UNIVERSE")

primary_model_universe = [
    {
        "model_id": "STAGE16_XGBOOST_TUNED",
        "family": "XGBoost",
        "comparison_group": "GROUP_A_DUPSAFE70",
        "profile_role": "PRIMARY",
        "deployment_unit": "single_model",
    },
    {
        "model_id": "STAGE16_LIGHTGBM_TUNED",
        "family": "LightGBM",
        "comparison_group": "GROUP_A_DUPSAFE70",
        "profile_role": "PRIMARY",
        "deployment_unit": "single_model",
    },
    {
        "model_id": "STAGE16_CATBOOST_TUNED",
        "family": "CatBoost",
        "comparison_group": "GROUP_A_DUPSAFE70",
        "profile_role": "PRIMARY",
        "deployment_unit": "single_model",
    },
    {
        "model_id": "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
        "family": "FT-Transformer",
        "comparison_group": "GROUP_A_DUPSAFE70",
        "profile_role": "PRIMARY",
        "deployment_unit": "five_checkpoint_soft_voting",
    },
    {
        "model_id": "STAGE20_MASKED_CNN_V1",
        "family": "CNN",
        "comparison_group": "GROUP_B_PACKET_IMAGE",
        "profile_role": "PRIMARY",
        "deployment_unit": "single_model",
    },
    {
        "model_id": "STAGE21_MASKED_VIT_V1",
        "family": "ViT",
        "comparison_group": "GROUP_B_PACKET_IMAGE",
        "profile_role": "PRIMARY",
        "deployment_unit": "single_model",
    },
]

primary_df = pd.DataFrame(primary_model_universe)

print(primary_df.to_string(index=False))


# =============================================================================
# 9. NON-PRIMARY / EXCLUDED DISCOVERIES
# =============================================================================

banner("STAGE26-0B :: NON-PRIMARY DISCOVERED FAMILIES")

exclusions = [
    {
        "family": "MLP",
        "stage26_primary": False,
        "reason": (
            "Discovered during 0A but not part of the predeclared "
            "Stage26 six-family profiling plan. Do not add after discovery "
            "without a separate pre-measurement amendment."
        ),
    },
    {
        "family": "Graph Transformer",
        "stage26_primary": False,
        "reason": (
            "Different representation and scientific population; cannot be "
            "placed on either primary Stage26 frontier without a separately "
            "defined comparable protocol."
        ),
    },
    {
        "family": "M-Temporal / temporal neural models",
        "stage26_primary": False,
        "reason": (
            "Different temporal representation and experiment family; "
            "not part of the frozen Stage26 architecture set."
        ),
    },
    {
        "family": "Other Stage16 classical candidates",
        "stage26_primary": False,
        "reason": (
            "Not part of the predeclared Stage26 XGB/LGBM/CatBoost "
            "architecture subset."
        ),
    },
]

exclusion_df = pd.DataFrame(exclusions)

print(exclusion_df.to_string(index=False))


# =============================================================================
# 10. COMPARABILITY GROUP DEFINITIONS
# =============================================================================

banner("STAGE26-0B :: COMPARABILITY GROUPS")

groups = {
    "GROUP_A_DUPSAFE70": {
        "name": "Duplicate-safe 70-feature tabular / FT comparison",
        "members": [
            "STAGE16_XGBOOST_TUNED",
            "STAGE16_LIGHTGBM_TUNED",
            "STAGE16_CATBOOST_TUNED",
            "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
        ],
        "predictor_count": 70,
        "population_semantics": (
            "Stage15 duplicate-safe train/validation/holdout partition "
            "inherited exactly by Stage16 classical benchmark."
        ),
        "primary_pareto_allowed": True,
        "cross_group_pareto_allowed": False,
        "metric_source_rule": (
            "Only discrimination values evaluated on the same frozen "
            "population may share a Pareto frontier."
        ),
    },

    "GROUP_B_PACKET_IMAGE": {
        "name": "Stage20/21 packet-image CNN vs ViT",
        "members": [
            "STAGE20_MASKED_CNN_V1",
            "STAGE21_MASKED_VIT_V1",
        ],
        "representation": "64x256x1_FROZEN_STAGE20_PACKET_IMAGE",
        "friday_rows": 12088,
        "friday_benign": 6486,
        "friday_attack": 5602,
        "population_semantics": (
            "Exact same Friday compact corpus export order; "
            "locked reuse benchmark."
        ),
        "primary_pareto_allowed": True,
        "cross_group_pareto_allowed": False,
        "claim_boundary": (
            "DESCRIPTIVE / NON-CONFIRMATORY ARCHITECTURE BENCHMARK"
        ),
    },
}

for group_id, spec in groups.items():
    print("\n" + group_id)
    print("-" * len(group_id))

    for key, value in spec.items():
        print(f"{key}: {value}")


# =============================================================================
# 11. OPERATIONAL REFERENCES
# =============================================================================

banner("STAGE26-0B :: OPERATIONAL REFERENCES")

operational_references = {
    "ENS_LGBM_XGB_EQUAL": {
        "role": "FINAL_STAGE16_OPERATIONAL_CLASSICAL_STRATEGY",
        "members": ["STAGE16_LIGHTGBM_TUNED", "STAGE16_XGBOOST_TUNED"],
        "weights": [0.5, 0.5],
        "threshold": final_strategy["operating_threshold"],
        "holdout_pr_auc": final_strategy["holdout_metrics"]["pr_auc"],
        "profile_in_stage26": True,
        "count_as_architecture_family": False,
        "reason": (
            "Stage16 final frozen operational strategy. Profile its actual "
            "deployment cost as an operational reference, but do not use it "
            "to inflate the six-family architecture candidate count."
        ),
    },

    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE": {
        "role": "RESOURCE_DECOMPOSITION_ONLY",
        "representative_seed": FT_RESOURCE_REPRESENTATIVE_SEED,
        "selection_rule": (
            "Lowest numerical seed in the already frozen five-checkpoint "
            "confirmation set; selected without consulting performance."
        ),
        "profile_in_stage26": True,
        "count_as_architecture_family": False,
        "eligible_for_predictive_pareto": False,
        "reason": (
            "Allows Stage26 to distinguish one FT forward-pass cost from "
            "the actual five-checkpoint ensemble deployment cost."
        ),
    },
}

print(
    json.dumps(
        operational_references,
        indent=2,
        sort_keys=True,
    )
)


# =============================================================================
# 12. ASSEMBLE ARTIFACT TABLE
# =============================================================================

banner("STAGE26-0B :: RESOLVED ARTIFACT TABLE")

resolved_records = (
    classical_records
    + ft_records
    + [cnn_record, vit_record]
)

resolved_df = pd.DataFrame(resolved_records)

display_columns = [
    "artifact_id",
    "family",
    "role",
    "comparison_group",
    "representation",
    "repo_relative_path",
    "size_bytes",
    "sha256_match",
    "size_match",
]

print(
    resolved_df[
        display_columns
    ].to_string(index=False)
)


# =============================================================================
# 13. SAVE STAGE26-0B OUTPUTS
# =============================================================================

banner("STAGE26-0B :: SAVING")

resolved_csv = (
    RUNTIME_DIR
    / "stage26_0b_resolved_artifacts.csv"
)

dependency_csv = (
    RUNTIME_DIR
    / "stage26_0b_required_dependencies.csv"
)

candidate_json = (
    PROTOCOL_DIR
    / "stage26_0b_candidate_registry.json"
)

group_json = (
    PROTOCOL_DIR
    / "stage26_0b_comparability_groups.json"
)

resolution_json = (
    PROTOCOL_DIR
    / "stage26_0b_resolution_record.json"
)


resolved_df.to_csv(
    resolved_csv,
    index=False,
)

pd.DataFrame(
    dependency_records
).to_csv(
    dependency_csv,
    index=False,
)


candidate_payload = {
    "schema": "stage26_candidate_registry_v1",
    "stage": 26,
    "repository_head": HEAD,
    "status": "RESOLVED_NOT_YET_MEASUREMENT_FROZEN",
    "primary_model_universe": primary_model_universe,
    "operational_references": operational_references,
    "excluded_discovered_families": exclusions,
}

candidate_json.write_text(
    json.dumps(
        candidate_payload,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


group_json.write_text(
    json.dumps(
        {
            "schema": "stage26_comparability_groups_v1",
            "stage": 26,
            "repository_head": HEAD,
            "groups": groups,
            "global_rule": (
                "NO CROSS-GROUP PARETO FRONTIER. "
                "Discrimination values from different frozen populations "
                "must not be plotted as directly comparable."
            ),
        },
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


resolution_payload = {
    "schema": "stage26_0b_resolution_record_v1",
    "stage": 26,
    "repository_head": HEAD,

    "scientific_status": (
        "ARTIFACT_AND_COMPARABILITY_RESOLUTION_COMPLETE"
    ),

    "ft_transformer": {
        "candidate_id": stage15_arch["candidate_id"],
        "input_predictor_count": stage15_arch["input_predictor_count"],
        "architecture": stage15_arch["architecture"],
        "final_predictor": primary_ensemble["model"],
        "checkpoint_seeds": primary_ensemble["checkpoint_seeds"],
        "ensemble_method": primary_ensemble["ensemble_method"],
        "resource_representative_seed": FT_RESOURCE_REPRESENTATIVE_SEED,
    },

    "stage16": {
        "comparability_decision": stage16_cmp["audit_decision"],
        "primary_classical_ids": PRIMARY_CLASSICAL_IDS,
        "final_operational_strategy": final_strategy["strategy_id"],
    },

    "stage20_stage21": {
        "representation": representation,
        "friday_population": friday_population,
        "comparison_role": stage21_cmp["role"],
    },

    "design_boundary": {
        "measurement_protocol_frozen": False,
        "models_deserialized": False,
        "models_loaded": False,
        "inference_executed": False,
        "latency_measured": False,
        "memory_profiled": False,
        "holdout_reopened": False,
        "gpu_used": False,
    },
}

resolution_json.write_text(
    json.dumps(
        resolution_payload,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


print("Resolved artifact table:")
print(" ", resolved_csv)

print("\nRequired dependencies:")
print(" ", dependency_csv)

print("\nCandidate registry:")
print(" ", candidate_json)

print("\nComparability groups:")
print(" ", group_json)

print("\nResolution record:")
print(" ", resolution_json)


# =============================================================================
# 14. FINAL GATES
# =============================================================================

banner("STAGE26-0B FINAL GATES")

# All upstream hashed artifacts where an expected hash exists must match.
expected_hash_rows = resolved_df[
    resolved_df["expected_sha256"].notna()
]

bad_hashes = expected_hash_rows[
    expected_hash_rows["sha256_match"] != True
]

bad_sizes = expected_hash_rows[
    expected_hash_rows["size_match"] != True
]

print("Expected-hash artifacts :", len(expected_hash_rows))
print("Hash mismatches          :", len(bad_hashes))
print("Size mismatches          :", len(bad_sizes))

if len(bad_hashes):
    print(bad_hashes[
        ["artifact_id", "repo_relative_path", "sha256_match"]
    ].to_string(index=False))

    raise RuntimeError(
        "Stage26-0B failed: upstream artifact hash mismatch."
    )

if len(bad_sizes):
    print(bad_sizes[
        ["artifact_id", "repo_relative_path", "size_match"]
    ].to_string(index=False))

    raise RuntimeError(
        "Stage26-0B failed: upstream artifact size mismatch."
    )


# =============================================================================
# 15. CLOSURE
# =============================================================================

banner("STAGE26-0B COMPLETE")

print(
    "Primary architecture families resolved:\n"
    "  1. XGBoost\n"
    "  2. LightGBM\n"
    "  3. CatBoost\n"
    "  4. FT-Transformer\n"
    "  5. CNN\n"
    "  6. ViT\n"
)

print(
    "Comparable populations:\n"
    "  GROUP_A_DUPSAFE70\n"
    "    XGBoost / LightGBM / CatBoost / FT-Transformer\n"
    "\n"
    "  GROUP_B_PACKET_IMAGE\n"
    "    Stage20 CNN / Stage21 ViT\n"
)

print(
    "Operational references:\n"
    "  ENS_LGBM_XGB_EQUAL\n"
    "  FT_BALANCED_SINGLE_RESOURCE_REFERENCE\n"
)

print(
    "Scientific state:\n"
    "  EXACT ARTIFACTS RESOLVED\n"
    "  COMPARABILITY GROUPS RESOLVED\n"
    "  NO MODEL DESERIALIZED\n"
    "  NO MODEL LOADED\n"
    "  NO INFERENCE EXECUTED\n"
    "  NO HOLDOUT REOPENED\n"
    "  NO LATENCY MEASURED\n"
    "  NO MEMORY PROFILED\n"
    "  GPU NOT USED\n"
    "  MEASUREMENT PROTOCOL NOT YET FROZEN\n"
)

print(
    "NEXT:\n"
    "  STAGE26-0C — freeze CPU benchmark semantics, execution paths,\n"
    "  batch semantics, warmup/timed iterations, process isolation,\n"
    "  thread/affinity policy, raw timing retention, bootstrap CIs,\n"
    "  and the later GPU-session handoff contract.\n"
    "\n"
    "  Only after 0C is committed may CPU profiling begin."
)


STAGE26-0B :: REPOSITORY GATE
Expected HEAD : f9307d5f7a4c8fd53886d66b3e5f8e74a65d87c6
Actual HEAD   : f9307d5f7a4c8fd53886d66b3e5f8e74a65d87c6
Clean repo    : True

STAGE26-0B :: AUTHORITATIVE UPSTREAM LOCKS
[OK] results/stage15_transformer_checkpoint/stage15_4c_frozen_architecture.json  sha256=4a1f5788eaacaca5...
[OK] results/stage15_transformer_checkpoint/stage15_5b_preholdout_decision_lock.json  sha256=eceeb4f7d5df4bfc...
[OK] results/stage15_transformer_checkpoint/stage15_6a_holdout_evaluation_result.json  sha256=502baef03425921b...
[OK] results/stage16_classical_benchmark_checkpoint/stage16_0_classical_comparability_audit.json  sha256=0def006ab5e0a28f...
[OK] results/stage16_classical_benchmark_checkpoint/stage16_3b_tuned_model_manifest.json  sha256=d1e6f385c0fb10b5...
[OK] results/stage16_classical_benchmark_checkpoint/stage16_6c_final_classical_holdout_lock.json  sha256=8eafe1bedb8c5fed...
[OK] results/stage21_architecture/stage21_0_cnn_vit_followup_protocol_lock.json  sha256=

In [6]:
# =============================================================================
# STAGE26-0C — DEPLOYMENT / RESOURCE MEASUREMENT PROTOCOL FREEZE
#
# STATUS AFTER SUCCESS
# --------------------
# FROZEN BEFORE FIRST STAGE26 RESOURCE MEASUREMENT
#
# THIS CELL:
#   - verifies all upstream Stage26-0A/0B protocol artifacts
#   - closes CNN + ViT model-state provenance
#   - freezes CPU topology / affinity semantics
#   - freezes thread controls
#   - freezes deterministic synthetic benchmark-input semantics
#   - freezes cold-start semantics
#   - freezes warm latency / throughput semantics
#   - freezes memory measurement semantics
#   - freezes batch sizes and iteration counts
#   - freezes environmental contamination rules
#   - freezes raw timing retention
#   - freezes bootstrap CI procedure
#   - freezes comparable-population Pareto rules
#   - freezes failure / OOM / timeout handling
#   - freezes later GPU-session handoff rules
#   - creates deterministic randomized CPU execution plan
#   - creates durable Stage26-0 protocol package inside the repository
#
# THIS CELL DOES NOT:
#   - deserialize any model
#   - execute any model
#   - perform inference
#   - measure latency
#   - measure memory
#   - access predictive holdout rows
#   - use a GPU
#
# CRITICAL:
#   After this cell succeeds, DO NOT start profiling yet.
#   First git-anchor and push this protocol package.
# =============================================================================

from __future__ import annotations

import os
import sys
import json
import math
import shutil
import hashlib
import platform
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import psutil


# =============================================================================
# 0. CONFIG
# =============================================================================

STAGE = 26

EXPECTED_HEAD = "f9307d5f7a4c8fd53886d66b3e5f8e74a65d87c6"

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

RUNTIME_DIR = STAGE26_ROOT / "runtime"
PROTOCOL_DIR = STAGE26_ROOT / "protocol"

REPO_STAGE26_ROOT = (
    REPO_DIR
    / "results"
    / "stage26_deployment_profiling"
)

REPO_PROTOCOL_DIR = (
    REPO_STAGE26_ROOT
    / "stage26_0_protocol_lock"
)

REPO_PROTOCOL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Deterministic Stage26 measurement seed.
MEASUREMENT_SEED = 26042

# Bootstrap seed is deliberately the same frozen Stage26 seed.
BOOTSTRAP_SEED = MEASUREMENT_SEED

# Primary inference batch sizes from the Stage26 design.
BATCH_SIZES = [
    1,
    64,
    256,
    1024,
    8192,
]

# Frozen warmup / timed-repeat counts.
#
# Tail-latency emphasis is strongest at batch=1.
# Larger throughput-oriented batches use fewer iterations to keep CPU
# profiling tractable while still retaining repeated raw measurements.
ITERATION_POLICY = {
    1: {
        "warmup_runs": 50,
        "timed_runs": 200,
        "role": "LATENCY_PRIMARY",
    },
    64: {
        "warmup_runs": 30,
        "timed_runs": 150,
        "role": "MICROBATCH",
    },
    256: {
        "warmup_runs": 20,
        "timed_runs": 100,
        "role": "MICROBATCH",
    },
    1024: {
        "warmup_runs": 10,
        "timed_runs": 50,
        "role": "THROUGHPUT",
    },
    8192: {
        "warmup_runs": 5,
        "timed_runs": 20,
        "role": "THROUGHPUT_STRESS",
    },
}

# Cold-start repetitions use fresh Python interpreter processes.
COLD_START_REPETITIONS = 20

# Memory measurements occur separately from timing.
MEMORY_REPETITIONS = 5

# Process RSS sampler interval for memory-only runs.
RSS_SAMPLE_INTERVAL_MS = 5

# Bootstrap uncertainty.
BOOTSTRAP_REPLICATES = 2000
BOOTSTRAP_CI_PERCENT = 95.0

# Environmental controls.
CPU_UTILIZATION_GATE_PERCENT = 20.0
MIN_AVAILABLE_RAM_GIB = 8.0
ENVIRONMENT_RETRY_COUNT = 3
ENVIRONMENT_RETRY_COOLDOWN_SECONDS = 5

# Measurement condition isolation.
CONDITION_COOLDOWN_SECONDS = 2

# Hard subprocess ceiling.
#
# If a condition exceeds this:
#   => TIMEOUT_RESOURCE_LIMIT
# and we DO NOT change its batch size or iteration policy post hoc.
CONDITION_TIMEOUT_SECONDS = 600

# Memory safety.
#
# Worker subprocesses must not intentionally allocate beyond this proportion
# of total physical RAM. Unsupported conditions are reported, not adapted.
MAX_WORKER_RAM_FRACTION = 0.85


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text: str):
    print("\n" + "=" * 104)
    print(text)
    print("=" * 104)


def git(*args):
    p = subprocess.run(
        ["git", *args],
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
):
    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def sha256_text(text: str):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def canonical_json_bytes(payload):
    return (
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            separators=(",", ": "),
            ensure_ascii=False,
        )
        + "\n"
    ).encode("utf-8")


def write_canonical_json(
    path: Path,
    payload,
):
    raw = canonical_json_bytes(
        payload
    )

    path.write_bytes(raw)

    return hashlib.sha256(
        raw
    ).hexdigest()


def require_file(path: Path):
    if not path.exists():
        raise FileNotFoundError(
            path
        )

    if not path.is_file():
        raise RuntimeError(
            f"Expected regular file: {path}"
        )


def read_int_file(path: Path):
    try:
        return int(
            path.read_text().strip()
        )
    except Exception:
        return None


# =============================================================================
# 2. REPOSITORY + PREVIOUS STAGE26 GATES
# =============================================================================

banner(
    "STAGE26-0C :: REPOSITORY / UPSTREAM STAGE26 GATE"
)

HEAD = git(
    "rev-parse",
    "HEAD",
)

STATUS_BEFORE = git(
    "status",
    "--porcelain",
)

print("Expected HEAD :", EXPECTED_HEAD)
print("Actual HEAD   :", HEAD)
print("Clean before  :", STATUS_BEFORE == "")

if HEAD != EXPECTED_HEAD:
    raise RuntimeError(
        "Repository HEAD changed before Stage26-0C."
    )

if STATUS_BEFORE:
    raise RuntimeError(
        "Repository must be clean before creating "
        "the Stage26-0C protocol package:\n"
        + STATUS_BEFORE
    )


BOOTSTRAP_MANIFEST = (
    RUNTIME_DIR
    / "stage26_bootstrap_manifest.json"
)

ARTIFACT_INVENTORY = (
    RUNTIME_DIR
    / "stage26_artifact_inventory.csv"
)

INPUT_INVENTORY = (
    RUNTIME_DIR
    / "stage26_kaggle_input_inventory.csv"
)

RESOLVED_ARTIFACTS = (
    RUNTIME_DIR
    / "stage26_0b_resolved_artifacts.csv"
)

REQUIRED_DEPENDENCIES = (
    RUNTIME_DIR
    / "stage26_0b_required_dependencies.csv"
)

CANDIDATE_REGISTRY = (
    PROTOCOL_DIR
    / "stage26_0b_candidate_registry.json"
)

COMPARABILITY_GROUPS = (
    PROTOCOL_DIR
    / "stage26_0b_comparability_groups.json"
)

RESOLUTION_RECORD = (
    PROTOCOL_DIR
    / "stage26_0b_resolution_record.json"
)


UPSTREAM_STAGE26_FILES = [
    BOOTSTRAP_MANIFEST,
    ARTIFACT_INVENTORY,
    INPUT_INVENTORY,
    RESOLVED_ARTIFACTS,
    REQUIRED_DEPENDENCIES,
    CANDIDATE_REGISTRY,
    COMPARABILITY_GROUPS,
    RESOLUTION_RECORD,
]


print("\nUpstream Stage26 files:")

upstream_stage26_hashes = {}

for path in UPSTREAM_STAGE26_FILES:

    require_file(path)

    digest = sha256_file(path)

    upstream_stage26_hashes[
        path.name
    ] = digest

    print(
        f"[OK] {path.name:50s} "
        f"{digest[:20]}..."
    )


# =============================================================================
# 3. CLOSE CNN / VIT ARTIFACT PROVENANCE
# =============================================================================

banner(
    "STAGE26-0C :: CNN / VIT FINAL ARTIFACT IDENTITY GATE"
)


CNN_PATH = (
    REPO_DIR
    / "results/stage20_1e_training/"
      "stage20_1e2_epoch10_model_state_dict.pt"
)

VIT_PATH = (
    REPO_DIR
    / "results/stage21_architecture/"
      "stage21_2_epoch10_model_state_dict.pt"
)

ENCODER_PATH = (
    REPO_DIR
    / "scripts/stage20_packet_image_encoder.py"
)

FT_SCALER_PATH = (
    REPO_DIR
    / "results/stage15_transformer_checkpoint/"
      "stage15_2_standard_scaler.joblib"
)


for path in [
    CNN_PATH,
    VIT_PATH,
    ENCODER_PATH,
    FT_SCALER_PATH,
]:
    require_file(path)


# Frozen historical identities.
CNN_EXPECTED_SHA256 = (
    "3ebc71e579dc8e0e545981b2d60eea643148fe53e0902f8df8e47556243ad30b"
)

CNN_EXPECTED_SIZE = 376879

VIT_EXPECTED_SHA256 = (
    "221e9c805fb663acacf2f0f2ca95dba7cb4b2ec4c4de5a3650cf4adeb99b5ef8"
)

VIT_EXPECTED_SIZE = 378999

ENCODER_EXPECTED_SHA256 = (
    "9883fe2b27020aaff707a753123b35eb3223d21abf295d056ec233e532f94222"
)

FT_SCALER_EXPECTED_SHA256 = (
    "ac1e3a9b0a2409edcd98c293f31c22b9c140cc882518881565d16eb1e2a573b5"
)


identity_checks = [
    (
        "Stage20 CNN",
        CNN_PATH,
        CNN_EXPECTED_SHA256,
        CNN_EXPECTED_SIZE,
    ),
    (
        "Stage21 ViT",
        VIT_PATH,
        VIT_EXPECTED_SHA256,
        VIT_EXPECTED_SIZE,
    ),
    (
        "Stage20 encoder",
        ENCODER_PATH,
        ENCODER_EXPECTED_SHA256,
        None,
    ),
    (
        "Stage15 scaler",
        FT_SCALER_PATH,
        FT_SCALER_EXPECTED_SHA256,
        None,
    ),
]


artifact_identity_records = []

for (
    name,
    path,
    expected_sha,
    expected_size,
) in identity_checks:

    actual_sha = sha256_file(
        path
    )

    actual_size = (
        path.stat().st_size
    )

    sha_ok = (
        actual_sha
        ==
        expected_sha
    )

    size_ok = (
        True
        if expected_size is None
        else actual_size == expected_size
    )

    print(
        f"{name:18s} "
        f"sha={sha_ok} "
        f"size={size_ok} "
        f"bytes={actual_size}"
    )

    if not sha_ok:
        raise RuntimeError(
            f"{name} SHA256 mismatch."
        )

    if not size_ok:
        raise RuntimeError(
            f"{name} size mismatch."
        )

    artifact_identity_records.append(
        {
            "name": name,
            "repo_relative_path": str(
                path.relative_to(
                    REPO_DIR
                )
            ),
            "size_bytes": actual_size,
            "sha256": actual_sha,
        }
    )


# =============================================================================
# 4. CPU TOPOLOGY + AFFINITY FREEZE
# =============================================================================

banner(
    "STAGE26-0C :: CPU TOPOLOGY / AFFINITY FREEZE"
)


if hasattr(
    os,
    "sched_getaffinity",
):
    allowed_cpus = sorted(
        os.sched_getaffinity(
            0
        )
    )
else:
    allowed_cpus = list(
        range(
            os.cpu_count()
            or 1
        )
    )


cpu_topology = []

for cpu_id in allowed_cpus:

    topology_root = Path(
        f"/sys/devices/system/cpu/cpu{cpu_id}/topology"
    )

    package_id = read_int_file(
        topology_root
        / "physical_package_id"
    )

    core_id = read_int_file(
        topology_root
        / "core_id"
    )

    cpu_topology.append(
        {
            "logical_cpu": cpu_id,
            "physical_package_id": package_id,
            "core_id": core_id,
        }
    )


# Select one logical CPU from each unique physical core.
seen_physical = set()
physical_core_representatives = []

for row in cpu_topology:

    if (
        row["physical_package_id"]
        is None
        or
        row["core_id"]
        is None
    ):
        continue

    key = (
        row["physical_package_id"],
        row["core_id"],
    )

    if key not in seen_physical:
        seen_physical.add(
            key
        )

        physical_core_representatives.append(
            row["logical_cpu"]
        )


topology_resolution = (
    "SYSFS_PHYSICAL_CORE_MAPPING"
)

if len(
    physical_core_representatives
) < 2:

    # Fallback should not be required on the current Kaggle CPU runtime,
    # but is explicit and durable.
    physical_core_representatives = (
        allowed_cpus[:2]
    )

    topology_resolution = (
        "LOGICAL_CPU_FALLBACK"
    )


if not physical_core_representatives:
    raise RuntimeError(
        "Unable to resolve usable CPU affinity."
    )


PRIMARY_CPU_AFFINITY = [
    physical_core_representatives[
        0
    ]
]

SERVER_CPU_AFFINITY = (
    physical_core_representatives[
        :min(
            2,
            len(
                physical_core_representatives
            ),
        )
    ]
)


print("Allowed logical CPUs       :", allowed_cpus)
print("Physical representatives   :", physical_core_representatives)
print("Topology resolution        :", topology_resolution)
print("Primary 1-core affinity    :", PRIMARY_CPU_AFFINITY)
print("Secondary server affinity  :", SERVER_CPU_AFFINITY)

print("\nTopology table:")

for row in cpu_topology:
    print(
        f"  logical={row['logical_cpu']} "
        f"package={row['physical_package_id']} "
        f"core={row['core_id']}"
    )


# =============================================================================
# 5. THREAD POLICY
# =============================================================================

banner(
    "STAGE26-0C :: THREAD POLICY"
)


THREAD_ENVIRONMENT_KEYS = [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "BLIS_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
]


CPU_EXECUTION_MODES = {
    "CPU_1_PHYSICAL_CORE": {
        "role": "PRIMARY_CPU_EFFICIENCY_BASELINE",
        "thread_count": 1,
        "affinity": PRIMARY_CPU_AFFINITY,
        "batch_sizes": BATCH_SIZES,
    },

    "CPU_2_PHYSICAL_CORE": {
        "role": "SECONDARY_SERVER_THROUGHPUT_CONDITION",
        "thread_count": len(
            SERVER_CPU_AFFINITY
        ),
        "affinity": SERVER_CPU_AFFINITY,
        "batch_sizes": BATCH_SIZES,
    },
}


print(
    json.dumps(
        CPU_EXECUTION_MODES,
        indent=2,
        sort_keys=True,
    )
)


# =============================================================================
# 6. PRIMARY PROFILE TARGETS
# =============================================================================

banner(
    "STAGE26-0C :: PROFILE TARGET FREEZE"
)


PROFILE_TARGETS = [
    {
        "target_id": "STAGE16_XGBOOST_TUNED",
        "family": "XGBoost",
        "role": "PRIMARY_ARCHITECTURE",
        "comparison_group": "GROUP_A_DUPSAFE70",
        "deployment_unit": "single_model",
    },
    {
        "target_id": "STAGE16_LIGHTGBM_TUNED",
        "family": "LightGBM",
        "role": "PRIMARY_ARCHITECTURE",
        "comparison_group": "GROUP_A_DUPSAFE70",
        "deployment_unit": "single_model",
    },
    {
        "target_id": "STAGE16_CATBOOST_TUNED",
        "family": "CatBoost",
        "role": "PRIMARY_ARCHITECTURE",
        "comparison_group": "GROUP_A_DUPSAFE70",
        "deployment_unit": "single_model",
    },
    {
        "target_id": "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
        "family": "FT-Transformer",
        "role": "PRIMARY_ARCHITECTURE",
        "comparison_group": "GROUP_A_DUPSAFE70",
        "deployment_unit": "five_checkpoint_soft_voting",
    },
    {
        "target_id": "STAGE20_MASKED_CNN_V1",
        "family": "CNN",
        "role": "PRIMARY_ARCHITECTURE",
        "comparison_group": "GROUP_B_PACKET_IMAGE",
        "deployment_unit": "single_model",
    },
    {
        "target_id": "STAGE21_MASKED_VIT_V1",
        "family": "ViT",
        "role": "PRIMARY_ARCHITECTURE",
        "comparison_group": "GROUP_B_PACKET_IMAGE",
        "deployment_unit": "single_model",
    },

    # Operational references: profiled, but not architecture-family additions.
    {
        "target_id": "ENS_LGBM_XGB_EQUAL",
        "family": "Classical Ensemble",
        "role": "OPERATIONAL_REFERENCE",
        "comparison_group": "GROUP_A_DUPSAFE70",
        "deployment_unit": "equal_weight_two_model_soft_voting",
    },
    {
        "target_id": "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
        "family": "FT-Transformer",
        "role": "RESOURCE_DECOMPOSITION_REFERENCE",
        "comparison_group": "GROUP_A_DUPSAFE70",
        "deployment_unit": "single_checkpoint_seed7",
    },
]


for target in PROFILE_TARGETS:
    print(
        f"{target['target_id']:43s} "
        f"{target['role']}"
    )


# =============================================================================
# 7. BENCHMARK INPUT SEMANTICS
# =============================================================================

banner(
    "STAGE26-0C :: BENCHMARK INPUT SEMANTICS"
)


INPUT_PROTOCOL = {
    "general": {
        "labels_used": False,
        "holdout_rows_used_for_model_timing": False,
        "data_dependent_model_selection": False,
        "input_generation_timed_as_inference": False,
        "reason": (
            "Model-only inference profiling must isolate execution cost "
            "from feature/preprocessing cost and must not reopen a "
            "predictive holdout solely for benchmarking."
        ),
    },

    "GROUP_A_DUPSAFE70": {
        "generator": "DETERMINISTIC_SCALER_GEOMETRY_SYNTHETIC",
        "rng": "numpy.random.default_rng_PCG64",
        "base_seed": MEASUREMENT_SEED,
        "predictor_count": 70,
        "dtype": "float32",

        "construction": {
            "scaled_matrix_Z": (
                "Z ~ Normal(0,1), deterministic from frozen seed "
                "derived from batch size."
            ),
            "tree_input_X_raw": (
                "X_raw = scaler.mean_ + Z * scaler.scale_. "
                "This preserves Stage15 training-feature scale "
                "without labels or holdout access."
            ),
            "ft_transformer_input": (
                "Use Z as the already-scaled FT-Transformer model input."
            ),
            "shared_rows_rule": (
                "At each batch size, trees and FT-Transformer derive "
                "their inputs from the exact same underlying Z rows."
            ),
        },

        "scaler": {
            "path": str(
                FT_SCALER_PATH.relative_to(
                    REPO_DIR
                )
            ),
            "sha256": FT_SCALER_EXPECTED_SHA256,
        },
    },

    "GROUP_B_PACKET_IMAGE": {
        "generator": "DETERMINISTIC_SYNTHETIC_IPV4_FLOW_ENCODER",
        "rng": "numpy.random.default_rng_PCG64",
        "base_seed": MEASUREMENT_SEED,
        "packet_image_rows": 64,
        "packet_image_cols": 256,
        "channels": 1,
        "model_dtype": "float32",

        "packet_recipe": {
            "packet_count_per_flow": (
                "Deterministically sampled integer in [1,64]."
            ),
            "packet_length_bytes": (
                "Deterministically sampled integer in [40,256]."
            ),
            "ip_version": 4,
            "ihl_bytes": 20,
            "protocol": (
                "Deterministic TCP/UDP mixture; protocol values 6 and 17."
            ),
            "fragment_offset": 0,
            "payload_bytes": (
                "Deterministic pseudo-random uint8 values."
            ),
        },

        "encoder": {
            "path": str(
                ENCODER_PATH.relative_to(
                    REPO_DIR
                )
            ),
            "sha256": ENCODER_EXPECTED_SHA256,
            "required_functions": [
                "mask_ipv4_packet",
                "encode_flow",
                "scale_for_model",
            ],
        },

        "model_boundary": {
            "image_scaling": "float32(image) / 255.0",
            "padding_mask_preserved": True,
            "exact_forward_inputs": (
                "Worker must use the original frozen architecture's "
                "required forward signature. If padding mask is required, "
                "it is passed exactly; if not required, it is not added."
            ),
        },
    },
}


print(
    json.dumps(
        INPUT_PROTOCOL,
        indent=2,
        sort_keys=True,
    )
)


# =============================================================================
# 8. TIMING SEMANTICS
# =============================================================================

banner(
    "STAGE26-0C :: TIMING SEMANTICS"
)


TIMING_PROTOCOL = {
    "clock": {
        "cpu": "time.perf_counter_ns",
        "raw_unit": "nanoseconds",
        "raw_values_retained": True,
    },

    "timed_boundary": {
        "T_preprocess": (
            "Required model-boundary transforms only. "
            "Measured separately from inference."
        ),

        "T_infer": (
            "Prepared model input -> materialized attack probability output."
        ),

        "T_decision": (
            "Threshold application is excluded from primary T_infer. "
            "It may be recorded separately as descriptive decision overhead."
        ),

        "T_end_to_end": (
            "T_extract + T_preprocess + T_infer, "
            "only where the extraction pipeline is scientifically compatible."
        ),
    },

    "cpu_inference_rules": {
        "torch": {
            "model_eval": True,
            "torch_inference_mode": True,
            "autograd": False,
            "input_prepared_before_timer": True,
        },

        "xgboost": {
            "training_forbidden": True,
            "model_conversion_forbidden": True,
            "native_loaded_artifact_only": True,
        },

        "lightgbm": {
            "training_forbidden": True,
            "model_conversion_forbidden": True,
            "native_loaded_artifact_only": True,
        },

        "catboost": {
            "training_forbidden": True,
            "model_conversion_forbidden": True,
            "native_loaded_artifact_only": True,
        },
    },

    "ensemble_rules": {
        "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING": (
            "Primary FT latency includes all five checkpoint forward passes "
            "plus unweighted arithmetic probability averaging."
        ),

        "ENS_LGBM_XGB_EQUAL": (
            "Operational classical ensemble latency includes both "
            "LightGBM and XGBoost probability generation plus equal-weight "
            "probability averaging."
        ),

        "FT_BALANCED_SINGLE_RESOURCE_REFERENCE": (
            "Single seed-7 checkpoint only; resource decomposition only; "
            "not eligible for predictive Pareto claims."
        ),
    },
}


print(
    json.dumps(
        TIMING_PROTOCOL,
        indent=2,
        sort_keys=True,
    )
)


# =============================================================================
# 9. COLD-START SEMANTICS
# =============================================================================

banner(
    "STAGE26-0C :: COLD-START SEMANTICS"
)


COLD_START_PROTOCOL = {
    "repetitions": COLD_START_REPETITIONS,

    "process_semantics": (
        "Each repetition launches a fresh Python interpreter using "
        "subprocess exec semantics. multiprocessing fork reuse is forbidden."
    ),

    "components": [
        "process_spawn_to_worker_ready_ms",
        "framework_import_ms",
        "model_deserialization_load_ms",
        "first_prediction_ms",
        "spawn_to_first_output_ms",
    ],

    "primary_cold_condition": "CPU_1_PHYSICAL_CORE",

    "input_policy": (
        "A frozen batch-size-1 deterministic Stage26 synthetic input "
        "is used for first prediction."
    ),

    "jit_policy": (
        "Any first-call framework/JIT initialization belongs to "
        "first_prediction_ms and cold total. It is excluded only from "
        "warm steady-state timing after frozen warmup."
    ),
}


print(
    json.dumps(
        COLD_START_PROTOCOL,
        indent=2,
        sort_keys=True,
    )
)


# =============================================================================
# 10. WARM STEADY-STATE SEMANTICS
# =============================================================================

banner(
    "STAGE26-0C :: WARM STEADY-STATE SEMANTICS"
)


WARM_PROTOCOL = {
    "batch_sizes": BATCH_SIZES,
    "iteration_policy": ITERATION_POLICY,

    "primary_latency_condition": {
        "batch_size": 1,
        "hardware_mode": "CPU_1_PHYSICAL_CORE",
        "tail_statistics": [
            "p50",
            "p95",
            "p99",
        ],
    },

    "throughput_metrics": [
        "batch_latency_seconds",
        "amortized_latency_per_flow_seconds",
        "flows_per_second",
    ],

    "reported_distribution_metrics": [
        "mean",
        "standard_deviation",
        "coefficient_of_variation",
        "p50",
        "p95",
        "p99_if_timed_sample_count_gte_100",
        "maximum_descriptive_only",
    ],

    "p99_policy": (
        "p99 is a primary tail statistic only for conditions with at least "
        "100 timed observations. It may not be promoted from the 20- or "
        "50-sample large-batch conditions."
    ),

    "cooldown_seconds_between_conditions": (
        CONDITION_COOLDOWN_SECONDS
    ),

    "condition_timeout_seconds": (
        CONDITION_TIMEOUT_SECONDS
    ),
}


print(
    json.dumps(
        WARM_PROTOCOL,
        indent=2,
        sort_keys=True,
    )
)


# =============================================================================
# 11. MEMORY SEMANTICS
# =============================================================================

banner(
    "STAGE26-0C :: MEMORY SEMANTICS"
)


MEMORY_PROTOCOL = {
    "timing_and_memory_runs_separate": True,

    "repetitions": MEMORY_REPETITIONS,

    "fresh_process_per_repetition": True,

    "rss_sampling_interval_ms": (
        RSS_SAMPLE_INTERVAL_MS
    ),

    "cpu_metrics": {
        "baseline_rss": (
            "RSS after required framework imports and input preparation, "
            "before model load."
        ),

        "loaded_rss": (
            "RSS after model load and before inference."
        ),

        "peak_rss": (
            "Maximum sampled process RSS during memory-only inference run."
        ),

        "ru_maxrss": (
            "Recorded descriptively inside the isolated process."
        ),

        "delta_model_rss": (
            "loaded_rss - baseline_rss"
        ),

        "delta_peak_rss": (
            "peak_rss - baseline_rss"
        ),
    },

    "package_size": {
        "serialized_model_size": (
            "Actual immutable model/checkpoint bytes on disk."
        ),

        "deployment_package_size": (
            "Sum of model/checkpoint artifacts plus model-specific "
            "required preprocessing/configuration/architecture artifacts. "
            "Generic Python/framework installation size is excluded."
        ),
    },

    "max_worker_ram_fraction": (
        MAX_WORKER_RAM_FRACTION
    ),
}


print(
    json.dumps(
        MEMORY_PROTOCOL,
        indent=2,
        sort_keys=True,
    )
)


# =============================================================================
# 12. ENVIRONMENTAL CONTAMINATION / FAILURE RULES
# =============================================================================

banner(
    "STAGE26-0C :: ENVIRONMENT / FAILURE RULES"
)


FAILURE_PROTOCOL = {
    "precondition_gate": {
        "cpu_utilization_percent_max": (
            CPU_UTILIZATION_GATE_PERCENT
        ),

        "cpu_utilization_sampling_seconds": 1.0,

        "available_ram_gib_min": (
            MIN_AVAILABLE_RAM_GIB
        ),

        "retry_count": (
            ENVIRONMENT_RETRY_COUNT
        ),

        "retry_cooldown_seconds": (
            ENVIRONMENT_RETRY_COOLDOWN_SECONDS
        ),

        "if_gate_never_passes": (
            "INVALID_ENVIRONMENT; do not silently benchmark."
        ),
    },

    "resource_failure": {
        "out_of_memory": (
            "Record RESOURCE_LIMIT_OOM. "
            "Do not reduce batch size post hoc for that condition."
        ),

        "timeout": (
            f"Record TIMEOUT_RESOURCE_LIMIT after "
            f"{CONDITION_TIMEOUT_SECONDS} seconds. "
            "Do not alter iteration counts post hoc."
        ),

        "backend_unavailable": (
            "Record BACKEND_UNAVAILABLE. "
            "Do not retrain, convert, or substitute another model."
        ),

        "serialization_incompatible": (
            "Block affected target. A narrowly scoped compatibility adapter "
            "may be written only if it does not change artifact bytes, "
            "model parameters, input representation, or measurement policy."
        ),
    },

    "no_post_result_adaptation": [
        "no model substitution",
        "no threshold retuning",
        "no architecture change",
        "no batch-size replacement",
        "no selective warmup extension",
        "no selective iteration extension",
        "no selective thread-count change",
        "no dropping slow models because they are slow",
    ],
}


print(
    json.dumps(
        FAILURE_PROTOCOL,
        indent=2,
        sort_keys=True,
    )
)


# =============================================================================
# 13. STATISTICAL SUMMARY / BOOTSTRAP FREEZE
# =============================================================================

banner(
    "STAGE26-0C :: STATISTICAL SUMMARY FREEZE"
)


STATISTICS_PROTOCOL = {
    "bootstrap": {
        "enabled": True,
        "replicates": (
            BOOTSTRAP_REPLICATES
        ),
        "seed": (
            BOOTSTRAP_SEED
        ),
        "rng": (
            "numpy.random.default_rng_PCG64"
        ),
        "interval": (
            f"PERCENTILE_{BOOTSTRAP_CI_PERCENT:.0f}_PERCENT"
        ),
    },

    "bootstrap_targets": [
        "p50_latency",
        "p95_latency",
        "p99_latency_when_n_gte_100",
        "median_throughput",
    ],

    "raw_timing_retention": {
        "required": True,
        "format": (
            "CSV plus optional NPY mirror"
        ),
        "minimum_columns": [
            "target_id",
            "hardware_mode",
            "batch_size",
            "iteration_index",
            "elapsed_ns",
            "flows_per_second",
        ],
    },

    "maximum_policy": (
        "Maximum latency is descriptive only and is not used "
        "for Pareto dominance."
    ),
}


print(
    json.dumps(
        STATISTICS_PROTOCOL,
        indent=2,
        sort_keys=True,
    )
)


# =============================================================================
# 14. PARETO / COMPARABILITY FREEZE
# =============================================================================

banner(
    "STAGE26-0C :: PARETO / COMPARABILITY FREEZE"
)


PARETO_PROTOCOL = {
    "global_rule": (
        "NO CROSS-GROUP PARETO FRONTIER."
    ),

    "GROUP_A_DUPSAFE70": {
        "eligible_primary_members": [
            "STAGE16_XGBOOST_TUNED",
            "STAGE16_LIGHTGBM_TUNED",
            "STAGE16_CATBOOST_TUNED",
            "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
        ],

        "primary_discrimination_axis": (
            "Same-population frozen PR-AUC"
        ),

        "primary_cost_axis": (
            "CPU_1_PHYSICAL_CORE batch1 p95 inference latency"
        ),
    },

    "GROUP_B_PACKET_IMAGE": {
        "eligible_primary_members": [
            "STAGE20_MASKED_CNN_V1",
            "STAGE21_MASKED_VIT_V1",
        ],

        "primary_discrimination_axis": (
            "Same-Friday-population frozen PR-AUC"
        ),

        "primary_cost_axis": (
            "CPU_1_PHYSICAL_CORE batch1 p95 inference latency"
        ),

        "claim_boundary": (
            "DESCRIPTIVE_NON_CONFIRMATORY"
        ),
    },

    "operational_references": {
        "ENS_LGBM_XGB_EQUAL": (
            "May be shown as operational reference but does not increase "
            "architecture-family candidate count."
        ),

        "FT_BALANCED_SINGLE_RESOURCE_REFERENCE": (
            "Resource decomposition only; excluded from predictive Pareto."
        ),
    },

    "dominance_rule": {
        "A_dominates_B_if": (
            "A PR-AUC >= B PR-AUC and A p95 latency <= B p95 latency, "
            "with at least one strict inequality."
        ),

        "frontier_status": (
            "Descriptive point-estimate frontier; bootstrap uncertainty "
            "reported separately."
        ),
    },
}


print(
    json.dumps(
        PARETO_PROTOCOL,
        indent=2,
        sort_keys=True,
    )
)


# =============================================================================
# 15. EXTRACTION SUBPROTOCOL GATE
# =============================================================================

banner(
    "STAGE26-0C :: FEATURE-EXTRACTION SUBPROTOCOL GATE"
)


EXTRACTION_PROTOCOL_GATE = {
    "status": "FROZEN_GATE_NOT_YET_EXECUTED",

    "inference_results_may_select_pcap": False,

    "pcap_required_before_extraction_timing": True,

    "required_before_first_extraction_measurement": [
        "exact PCAP source",
        "PCAP SHA256",
        "PCAP byte size",
        "packet count or exact packet bounds",
        "sampling rule",
        "disk_IO_included_boolean",
        "serialization_included_boolean",
        "JVM_startup_included_boolean",
        "Python_process_startup_included_boolean",
        "exact extractor versions",
    ],

    "metrics_required": [
        "packets_per_second",
        "bytes_per_second",
        "MiB_per_second",
        "flows_per_second",
    ],

    "scientific_rule": (
        "No extraction timing may begin until a separate "
        "stage26_extraction_protocol.json is frozen. "
        "That subprotocol may not be chosen using Stage26 latency results."
    ),
}


print(
    json.dumps(
        EXTRACTION_PROTOCOL_GATE,
        indent=2,
        sort_keys=True,
    )
)


# =============================================================================
# 16. LATER GPU SESSION HANDOFF CONTRACT
# =============================================================================

banner(
    "STAGE26-0C :: GPU HANDOFF CONTRACT"
)


GPU_HANDOFF_PROTOCOL = {
    "gpu_work_occurs_after_cpu_phase": True,

    "reason": (
        "Kaggle accelerator switch resets the session; CPU work must "
        "be fully persisted and git-anchored before enabling GPU."
    ),

    "gpu_session_requirements": [
        "fresh session bootstrap",
        "clone exact CPU-stage Git commit",
        "verify this measurement_protocol.json SHA256",
        "verify candidate registry SHA256",
        "verify model/checkpoint SHA256 values",
        "record exact GPU name",
        "record GPU UUID if available",
        "record driver version",
        "record CUDA runtime/build",
        "record framework versions",
    ],

    "gpu_batch_sizes": BATCH_SIZES,

    "gpu_iteration_policy": ITERATION_POLICY,

    "gpu_timing_rule": (
        "GPU work must be synchronized immediately before timer start "
        "and immediately before timer stop. Asynchronous launch timing "
        "without synchronization is invalid."
    ),

    "gpu_memory_rules": [
        "baseline allocated/reserved",
        "post-load allocated/reserved",
        "peak allocated during inference",
        "process-visible GPU memory where available",
    ],

    "backend_policy": (
        "Benchmark only already-valid officially supported GPU inference "
        "execution paths. If a frozen model family has no valid GPU inference "
        "path in the runtime, record BACKEND_UNAVAILABLE. "
        "Do not retrain or convert merely to force GPU support."
    ),

    "protocol_mutation_after_gpu_results": False,
}


print(
    json.dumps(
        GPU_HANDOFF_PROTOCOL,
        indent=2,
        sort_keys=True,
    )
)


# =============================================================================
# 17. BUILD DETERMINISTIC CPU EXECUTION PLAN
# =============================================================================

banner(
    "STAGE26-0C :: DETERMINISTIC CPU EXECUTION PLAN"
)


conditions = []

condition_counter = 0

for target in PROFILE_TARGETS:

    for hardware_mode, mode_spec in CPU_EXECUTION_MODES.items():

        for batch_size in mode_spec["batch_sizes"]:

            condition_counter += 1

            conditions.append(
                {
                    "condition_id": (
                        f"CPUCOND_{condition_counter:03d}"
                    ),
                    "target_id": target["target_id"],
                    "target_role": target["role"],
                    "comparison_group": target["comparison_group"],
                    "hardware_mode": hardware_mode,
                    "thread_count": mode_spec["thread_count"],
                    "affinity": mode_spec["affinity"],
                    "batch_size": batch_size,
                    "warmup_runs": (
                        ITERATION_POLICY[
                            batch_size
                        ]["warmup_runs"]
                    ),
                    "timed_runs": (
                        ITERATION_POLICY[
                            batch_size
                        ]["timed_runs"]
                    ),
                    "batch_role": (
                        ITERATION_POLICY[
                            batch_size
                        ]["role"]
                    ),
                }
            )


# Freeze deterministic randomized order.
rng = np.random.default_rng(
    MEASUREMENT_SEED
)

permutation = rng.permutation(
    len(
        conditions
    )
)


ordered_conditions = []

for execution_order, original_index in enumerate(
    permutation,
    start=1,
):

    condition = dict(
        conditions[
            int(
                original_index
            )
        ]
    )

    condition[
        "execution_order"
    ] = execution_order

    ordered_conditions.append(
        condition
    )


execution_plan = {
    "schema": "stage26_cpu_execution_plan_v1",
    "stage": 26,
    "measurement_seed": MEASUREMENT_SEED,
    "condition_count": len(
        ordered_conditions
    ),
    "randomization": (
        "numpy.random.default_rng(seed).permutation"
    ),
    "conditions": ordered_conditions,
}


print(
    "Frozen CPU conditions:",
    len(
        ordered_conditions
    )
)

print(
    "\nFirst 15 randomized conditions:"
)

for row in ordered_conditions[:15]:
    print(
        f"{row['execution_order']:03d} "
        f"{row['target_id']:42s} "
        f"{row['hardware_mode']:21s} "
        f"B={row['batch_size']:5d} "
        f"W={row['warmup_runs']:3d} "
        f"T={row['timed_runs']:3d}"
    )


# =============================================================================
# 18. CONSTRUCT MASTER MEASUREMENT PROTOCOL
# =============================================================================

banner(
    "STAGE26-0C :: MASTER PROTOCOL CONSTRUCTION"
)


protocol = {
    "schema": "stage26_measurement_protocol_v1",

    "stage": 26,

    "status": (
        "FROZEN_BEFORE_FIRST_STAGE26_RESOURCE_MEASUREMENT"
    ),

    "frozen_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),

    "repository_head_before_stage26_protocol_commit": HEAD,

    "scientific_success_criterion": (
        "Stage 26 succeeds when every frozen architecture receives a "
        "reproducible resource profile under the same declared hardware "
        "and measurement semantics, feature-extraction cost is explicitly "
        "separated from inference cost, and Pareto claims are restricted "
        "to scientifically comparable evaluation populations."
    ),

    "no_expected_conclusion_frozen": True,

    "hypotheses_not_protocol_truths": [
        "feature extraction may or may not dominate",
        "tree models may or may not dominate",
        "CNN may or may not be Pareto dominated",
        "ViT may or may not be Pareto dominated",
        "GPU may or may not materially improve deployment cost",
    ],

    "upstream_stage26_hashes": (
        upstream_stage26_hashes
    ),

    "artifact_identity_records": (
        artifact_identity_records
    ),

    "runtime_at_freeze": {
        "python": sys.version.replace(
            "\n",
            " ",
        ),

        "platform": platform.platform(),

        "physical_cpu_count_psutil": (
            psutil.cpu_count(
                logical=False
            )
        ),

        "logical_cpu_count_psutil": (
            psutil.cpu_count(
                logical=True
            )
        ),

        "total_ram_gib": (
            psutil.virtual_memory().total
            / 1024**3
        ),

        "gpu_present": False,
    },

    "cpu_topology": {
        "allowed_logical_cpus": allowed_cpus,
        "topology_resolution": topology_resolution,
        "topology_rows": cpu_topology,
        "physical_core_representatives": physical_core_representatives,
        "execution_modes": CPU_EXECUTION_MODES,
    },

    "thread_policy": {
        "environment_keys": THREAD_ENVIRONMENT_KEYS,

        "worker_rule": (
            "Every benchmark worker sets each supported thread environment "
            "variable to the frozen condition thread_count before importing "
            "numerical/ML frameworks."
        ),

        "backend_thread_rules": {
            "xgboost": "n_jobs = frozen thread_count where mutable for inference",
            "lightgbm": "num_threads = frozen thread count where mutable for inference",
            "catboost": "thread_count = frozen thread count for CPU prediction",
            "pytorch_intraop": "frozen thread count",
            "pytorch_interop": 1,
        },

        "smt_policy": (
            "Affinity uses one logical CPU per distinct physical core "
            "where sysfs topology is available."
        ),
    },

    "profile_targets": (
        PROFILE_TARGETS
    ),

    "input_protocol": (
        INPUT_PROTOCOL
    ),

    "timing_protocol": (
        TIMING_PROTOCOL
    ),

    "cold_start_protocol": (
        COLD_START_PROTOCOL
    ),

    "warm_protocol": (
        WARM_PROTOCOL
    ),

    "memory_protocol": (
        MEMORY_PROTOCOL
    ),

    "environment_and_failure_protocol": (
        FAILURE_PROTOCOL
    ),

    "statistics_protocol": (
        STATISTICS_PROTOCOL
    ),

    "pareto_protocol": (
        PARETO_PROTOCOL
    ),

    "extraction_protocol_gate": (
        EXTRACTION_PROTOCOL_GATE
    ),

    "gpu_handoff_protocol": (
        GPU_HANDOFF_PROTOCOL
    ),

    "measurement_order": {
        "cpu_execution_plan_seed": (
            MEASUREMENT_SEED
        ),

        "cpu_condition_count": (
            len(
                ordered_conditions
            )
        ),

        "order_randomized_before_results": True,
    },

    "scientific_boundary_at_freeze": {
        "models_deserialized": False,
        "models_loaded_for_stage26": False,
        "stage26_inference_calls": 0,
        "stage26_timing_observations": 0,
        "stage26_memory_measurements": 0,
        "stage26_extraction_measurements": 0,
        "holdout_reopened": False,
        "gpu_used": False,
    },

    "next_permitted_actions": [
        (
            "Create durable Stage26-0C git commit containing this "
            "protocol lock and execution plan."
        ),
        (
            "Push and verify the protocol commit remotely."
        ),
        (
            "Only after remote verification: run Stage26 CPU "
            "execution preflight without timing."
        ),
        (
            "Only after preflight passes: begin randomized CPU profiling."
        ),
    ],
}


# =============================================================================
# 19. WRITE LOCAL FROZEN PROTOCOL
# =============================================================================

banner(
    "STAGE26-0C :: WRITE FROZEN PROTOCOL"
)


LOCAL_PROTOCOL_PATH = (
    PROTOCOL_DIR
    / "measurement_protocol.json"
)

LOCAL_EXECUTION_PLAN_PATH = (
    PROTOCOL_DIR
    / "cpu_execution_plan.json"
)

LOCAL_FREEZE_RECORD_PATH = (
    PROTOCOL_DIR
    / "stage26_0c_protocol_freeze_record.json"
)


protocol_sha = write_canonical_json(
    LOCAL_PROTOCOL_PATH,
    protocol,
)

execution_plan_sha = write_canonical_json(
    LOCAL_EXECUTION_PLAN_PATH,
    execution_plan,
)


freeze_record = {
    "schema": "stage26_0c_protocol_freeze_record_v1",

    "stage": 26,

    "status": (
        "FROZEN_BEFORE_FIRST_RESOURCE_MEASUREMENT"
    ),

    "frozen_at_utc": (
        protocol[
            "frozen_at_utc"
        ]
    ),

    "repository_head": HEAD,

    "measurement_protocol": {
        "path": str(
            LOCAL_PROTOCOL_PATH
        ),
        "sha256": protocol_sha,
    },

    "cpu_execution_plan": {
        "path": str(
            LOCAL_EXECUTION_PLAN_PATH
        ),
        "sha256": execution_plan_sha,
    },

    "scientific_boundary": (
        protocol[
            "scientific_boundary_at_freeze"
        ]
    ),

    "next_action": (
        "GIT_ANCHOR_AND_REMOTE_VERIFY_BEFORE_ANY_MODEL_LOAD_OR_TIMING"
    ),
}


freeze_record_sha = write_canonical_json(
    LOCAL_FREEZE_RECORD_PATH,
    freeze_record,
)


print("measurement_protocol.json:")
print(" ", LOCAL_PROTOCOL_PATH)
print(" ", protocol_sha)

print("\ncpu_execution_plan.json:")
print(" ", LOCAL_EXECUTION_PLAN_PATH)
print(" ", execution_plan_sha)

print("\nfreeze record:")
print(" ", LOCAL_FREEZE_RECORD_PATH)
print(" ", freeze_record_sha)


# =============================================================================
# 20. BUILD DURABLE REPOSITORY PROTOCOL PACKAGE
# =============================================================================

banner(
    "STAGE26-0C :: BUILD REPOSITORY PROTOCOL PACKAGE"
)


# Inputs copied into the durable protocol package.
files_to_copy = [
    BOOTSTRAP_MANIFEST,
    ARTIFACT_INVENTORY,
    INPUT_INVENTORY,
    RESOLVED_ARTIFACTS,
    REQUIRED_DEPENDENCIES,
    CANDIDATE_REGISTRY,
    COMPARABILITY_GROUPS,
    RESOLUTION_RECORD,
    LOCAL_PROTOCOL_PATH,
    LOCAL_EXECUTION_PLAN_PATH,
    LOCAL_FREEZE_RECORD_PATH,
]


for source in files_to_copy:

    destination = (
        REPO_PROTOCOL_DIR
        / source.name
    )

    shutil.copy2(
        source,
        destination,
    )

    print(
        f"[COPIED] {source.name}"
    )


# =============================================================================
# 21. CREATE PACKAGE MANIFEST
# =============================================================================

banner(
    "STAGE26-0C :: PACKAGE MANIFEST"
)


package_files = []

for path in sorted(
    REPO_PROTOCOL_DIR.iterdir()
):

    if not path.is_file():
        continue

    if path.name == "stage26_0c_package_manifest.json":
        continue

    package_files.append(
        {
            "path": str(
                path.relative_to(
                    REPO_DIR
                )
            ),
            "size_bytes": (
                path.stat().st_size
            ),
            "sha256": (
                sha256_file(
                    path
                )
            ),
        }
    )


PACKAGE_MANIFEST_PATH = (
    REPO_PROTOCOL_DIR
    / "stage26_0c_package_manifest.json"
)


package_manifest = {
    "schema": "stage26_0c_package_manifest_v1",

    "stage": 26,

    "status": (
        "READY_FOR_GIT_ANCHOR_BEFORE_FIRST_MEASUREMENT"
    ),

    "repository_parent_head": HEAD,

    "measurement_protocol_sha256": (
        protocol_sha
    ),

    "cpu_execution_plan_sha256": (
        execution_plan_sha
    ),

    "protocol_freeze_record_sha256": (
        freeze_record_sha
    ),

    "file_count_excluding_manifest": (
        len(
            package_files
        )
    ),

    "files": (
        package_files
    ),

    "scientific_boundary": {
        "models_loaded": False,
        "inference_executed": False,
        "latency_measured": False,
        "memory_measured": False,
        "gpu_used": False,
    },
}


package_manifest_sha = write_canonical_json(
    PACKAGE_MANIFEST_PATH,
    package_manifest,
)


print(
    "Package manifest:"
)
print(
    " ",
    PACKAGE_MANIFEST_PATH,
)
print(
    " ",
    package_manifest_sha,
)


# =============================================================================
# 22. FINAL AUDIT
# =============================================================================

banner(
    "STAGE26-0C FINAL AUDIT"
)


# Re-read exact on-disk protocol and verify hash.
protocol_rehash = sha256_file(
    LOCAL_PROTOCOL_PATH
)

execution_plan_rehash = sha256_file(
    LOCAL_EXECUTION_PLAN_PATH
)


print(
    "Protocol SHA expected :",
    protocol_sha,
)
print(
    "Protocol SHA actual   :",
    protocol_rehash,
)

print(
    "\nExecution SHA expected:",
    execution_plan_sha,
)
print(
    "Execution SHA actual  :",
    execution_plan_rehash,
)


if protocol_sha != protocol_rehash:
    raise RuntimeError(
        "Protocol changed after write."
    )

if execution_plan_sha != execution_plan_rehash:
    raise RuntimeError(
        "Execution plan changed after write."
    )


STATUS_AFTER = git(
    "status",
    "--porcelain",
)

print(
    "\nRepository status after protocol package creation:"
)

print(
    STATUS_AFTER
    if STATUS_AFTER
    else "<CLEAN — unexpected, protocol package should be new>"
)


expected_stage26_prefix = (
    "?? results/stage26_deployment_profiling/"
)

if not STATUS_AFTER:
    raise RuntimeError(
        "Expected new Stage26 protocol package "
        "to make repository dirty."
    )


# Ensure no existing scientific files outside the new Stage26 result package
# were modified.
unexpected_changes = []

for line in STATUS_AFTER.splitlines():

    # Git porcelain format:
    # XY path
    path = line[3:]

    if not path.startswith(
        "results/stage26_deployment_profiling/"
    ):
        unexpected_changes.append(
            line
        )


if unexpected_changes:
    raise RuntimeError(
        "Unexpected repository changes outside Stage26:\n"
        + "\n".join(
            unexpected_changes
        )
    )


banner(
    "STAGE26-0C PROTOCOL FREEZE COMPLETE"
)


print(
    "MEASUREMENT PROTOCOL:"
)
print(
    f"  SHA256 = {protocol_sha}"
)

print(
    "\nCPU EXECUTION PLAN:"
)
print(
    f"  conditions = {len(ordered_conditions)}"
)
print(
    f"  SHA256     = {execution_plan_sha}"
)

print(
    "\nCPU MODES:"
)
print(
    "  PRIMARY   = 1 physical CPU core"
)
print(
    f"              affinity {PRIMARY_CPU_AFFINITY}"
)
print(
    "  SECONDARY = 2 physical CPU cores"
)
print(
    f"              affinity {SERVER_CPU_AFFINITY}"
)

print(
    "\nBATCH SIZES:"
)
print(
    " ",
    BATCH_SIZES,
)

print(
    "\nPRIMARY ARCHITECTURES:"
)
print(
    "  XGBoost"
)
print(
    "  LightGBM"
)
print(
    "  CatBoost"
)
print(
    "  FT-Transformer 5-checkpoint ensemble"
)
print(
    "  Stage20 CNN"
)
print(
    "  Stage21 ViT"
)

print(
    "\nOPERATIONAL REFERENCES:"
)
print(
    "  Equal-weight LightGBM + XGBoost ensemble"
)
print(
    "  Single FT seed-7 checkpoint resource reference"
)

print(
    "\nSCIENTIFIC STATE:"
)
print(
    "  PROTOCOL FROZEN"
)
print(
    "  EXECUTION ORDER FROZEN"
)
print(
    "  CNN IDENTITY VERIFIED"
)
print(
    "  VIT IDENTITY VERIFIED"
)
print(
    "  NO MODEL DESERIALIZED"
)
print(
    "  NO MODEL LOADED"
)
print(
    "  NO INFERENCE EXECUTED"
)
print(
    "  NO LATENCY MEASURED"
)
print(
    "  NO MEMORY MEASURED"
)
print(
    "  NO EXTRACTION TIMING PERFORMED"
)
print(
    "  HOLDOUT NOT REOPENED"
)
print(
    "  GPU NOT USED"
)

print(
    "\nCRITICAL NEXT ACTION:"
)
print(
    "  GIT-ANCHOR AND PUSH THIS STAGE26-0C PROTOCOL PACKAGE."
)
print(
    "  DO NOT LOAD OR BENCHMARK A MODEL BEFORE REMOTE VERIFICATION."
)


STAGE26-0C :: REPOSITORY / UPSTREAM STAGE26 GATE
Expected HEAD : f9307d5f7a4c8fd53886d66b3e5f8e74a65d87c6
Actual HEAD   : f9307d5f7a4c8fd53886d66b3e5f8e74a65d87c6
Clean before  : True

Upstream Stage26 files:
[OK] stage26_bootstrap_manifest.json                    9659e61ecd3f14ba512a...
[OK] stage26_artifact_inventory.csv                     b12f8ebe6d1b29f43945...
[OK] stage26_kaggle_input_inventory.csv                 32414f147f3cb027e529...
[OK] stage26_0b_resolved_artifacts.csv                  bb73dd1c63ab78942cea...
[OK] stage26_0b_required_dependencies.csv               3fae673a5e8c0ec6ef35...
[OK] stage26_0b_candidate_registry.json                 610172f5324d81dccf40...
[OK] stage26_0b_comparability_groups.json               44d54cfa97360a094a6c...
[OK] stage26_0b_resolution_record.json                  6f71ed6a347321a50df1...

STAGE26-0C :: CNN / VIT FINAL ARTIFACT IDENTITY GATE
Stage20 CNN        sha=True size=True bytes=376879
Stage21 ViT        sha=True size=True bytes=3

In [9]:
# =============================================================================
# STAGE26-0C-GIT — GIT ANCHOR + PUSH + REMOTE VERIFICATION
#
# PURPOSE
# -------
# Durably anchor the Stage26 measurement protocol BEFORE any model load,
# inference, latency observation, or memory profiling.
#
# THIS CELL:
#   - verifies the frozen protocol hashes
#   - verifies only Stage26 files are uncommitted
#   - commits the protocol package
#   - pushes to origin/main using Kaggle Secret authentication
#   - fetches origin/main back from GitHub
#   - verifies local commit == remote main
#   - verifies remote measurement_protocol.json SHA256
#   - verifies remote cpu_execution_plan.json SHA256
#   - verifies remote package manifest SHA256
#   - leaves the repository clean
#
# THIS CELL DOES NOT:
#   - load a model
#   - execute inference
#   - perform timing
#   - perform memory profiling
#   - use GPU
# =============================================================================

from __future__ import annotations

import os
import json
import stat
import hashlib
import tempfile
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient


# =============================================================================
# CONFIG
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

PROTOCOL_PACKAGE = (
    REPO_DIR
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
)

PROTOCOL_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/measurement_protocol.json"
)

PLAN_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/cpu_execution_plan.json"
)

FREEZE_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/stage26_0c_protocol_freeze_record.json"
)

PACKAGE_MANIFEST_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/stage26_0c_package_manifest.json"
)


EXPECTED_PARENT_HEAD = (
    "f9307d5f7a4c8fd53886d66b3e5f8e74a65d87c6"
)

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

EXPECTED_PLAN_SHA256 = (
    "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363"
)

EXPECTED_FREEZE_RECORD_SHA256 = (
    "95c1cc93c377d8f017de24e67f6fc04e192babdce4a32244df9741b5465191dc"
)

EXPECTED_PACKAGE_MANIFEST_SHA256 = (
    "620c4174db2df59ab178b7f11779ce442a4623df8faa34f9e4f0b0ccc484972e"
)

COMMIT_MESSAGE = (
    "stage26: freeze deployment resource measurement protocol"
)


# =============================================================================
# HELPERS
# =============================================================================

def banner(text: str):
    print("\n" + "=" * 100)
    print(text)
    print("=" * 100)


def run(
    cmd,
    *,
    cwd=REPO_DIR,
    env=None,
    check=True,
):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "$ " + " ".join(cmd) + "\n\n" + p.stdout
        )

    return p


def git(*args, env=None, check=True):
    return run(
        ["git", *args],
        env=env,
        check=check,
    ).stdout.strip()


def sha256_file(path: Path):
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def git_blob_bytes(ref: str, path: str):
    """
    Read exact bytes of a file from a git ref without checking it out.
    """
    p = subprocess.run(
        [
            "git",
            "show",
            f"{ref}:{path}",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stderr.decode(
                "utf-8",
                errors="replace",
            )
        )

    return p.stdout


def sha256_bytes(data: bytes):
    return hashlib.sha256(
        data
    ).hexdigest()


# =============================================================================
# 1. PRE-COMMIT SCIENTIFIC GATE
# =============================================================================

banner(
    "STAGE26-0C-GIT :: PRE-COMMIT GATE"
)

HEAD_BEFORE = git(
    "rev-parse",
    "HEAD",
)

STATUS_BEFORE = git(
    "status",
    "--porcelain",
)

print("Expected parent HEAD :", EXPECTED_PARENT_HEAD)
print("Actual parent HEAD   :", HEAD_BEFORE)

if HEAD_BEFORE != EXPECTED_PARENT_HEAD:
    raise RuntimeError(
        "Repository HEAD changed since Stage26-0C protocol freeze."
    )


if not STATUS_BEFORE:
    raise RuntimeError(
        "No uncommitted Stage26 protocol package found."
    )


print("\nUncommitted repository state:")
print(STATUS_BEFORE)


unexpected = []

for line in STATUS_BEFORE.splitlines():

    path = line[3:]

    if not path.startswith(
        "results/stage26_deployment_profiling/"
    ):
        unexpected.append(
            line
        )


if unexpected:
    raise RuntimeError(
        "Unexpected modifications exist outside Stage26:\n"
        + "\n".join(
            unexpected
        )
    )


print(
    "\n[PASS] Only Stage26 deployment-profiling files are uncommitted."
)


# =============================================================================
# 2. VERIFY FROZEN FILES BEFORE COMMIT
# =============================================================================

banner(
    "STAGE26-0C-GIT :: LOCAL FROZEN HASH VERIFICATION"
)


checks = [
    (
        PROTOCOL_REL,
        EXPECTED_PROTOCOL_SHA256,
    ),
    (
        PLAN_REL,
        EXPECTED_PLAN_SHA256,
    ),
    (
        FREEZE_REL,
        EXPECTED_FREEZE_RECORD_SHA256,
    ),
    (
        PACKAGE_MANIFEST_REL,
        EXPECTED_PACKAGE_MANIFEST_SHA256,
    ),
]


for relpath, expected_sha in checks:

    path = (
        REPO_DIR
        / relpath
    )

    if not path.exists():
        raise FileNotFoundError(
            path
        )

    actual_sha = sha256_file(
        path
    )

    ok = (
        actual_sha
        ==
        expected_sha
    )

    print(
        f"{Path(relpath).name:45s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual_sha}"
    )

    if not ok:
        raise RuntimeError(
            f"Frozen artifact changed before commit: {relpath}"
        )


# =============================================================================
# 3. GIT IDENTITY
# =============================================================================

banner(
    "STAGE26-0C-GIT :: GIT IDENTITY"
)


name = git(
    "config",
    "--get",
    "user.name",
    check=False,
)

email = git(
    "config",
    "--get",
    "user.email",
    check=False,
)


if not name:
    git(
        "config",
        "user.name",
        "themubasshir",
    )

    name = "themubasshir"


if not email:
    git(
        "config",
        "user.email",
        "themubasshir@users.noreply.github.com",
    )

    email = (
        "themubasshir@users.noreply.github.com"
    )


print("Git user.name :", name)
print("Git user.email:", email)


# =============================================================================
# 4. STAGE EXACT STAGE26 PACKAGE
# =============================================================================

banner(
    "STAGE26-0C-GIT :: STAGING"
)


git(
    "add",
    "--",
    "results/stage26_deployment_profiling",
)


staged = git(
    "diff",
    "--cached",
    "--name-status",
)


print(staged)


if not staged:
    raise RuntimeError(
        "Nothing was staged."
    )


for line in staged.splitlines():

    parts = line.split(
        "\t"
    )

    path = parts[-1]

    if not path.startswith(
        "results/stage26_deployment_profiling/"
    ):
        raise RuntimeError(
            "Unexpected staged file outside Stage26:\n"
            + line
        )


# =============================================================================
# 5. COMMIT
# =============================================================================

banner(
    "STAGE26-0C-GIT :: COMMIT"
)


commit_output = git(
    "commit",
    "-m",
    COMMIT_MESSAGE,
)

print(commit_output)


NEW_HEAD = git(
    "rev-parse",
    "HEAD",
)


print("\nParent commit :", EXPECTED_PARENT_HEAD)
print("Protocol commit:", NEW_HEAD)


if NEW_HEAD == EXPECTED_PARENT_HEAD:
    raise RuntimeError(
        "Commit did not advance repository HEAD."
    )


# Confirm parent relationship.
actual_parent = git(
    "rev-parse",
    f"{NEW_HEAD}^",
)


if actual_parent != EXPECTED_PARENT_HEAD:
    raise RuntimeError(
        "Stage26 protocol commit does not have the expected parent."
    )


# =============================================================================
# 6. RESOLVE KAGGLE GITHUB SECRET
# =============================================================================

banner(
    "STAGE26-0C-GIT :: GITHUB AUTH"
)


secret_client = UserSecretsClient()

SECRET_ALIASES = [
    "GITHUB_TOKEN",
    "github_token",
    "GH_TOKEN",
    "gh_token",
    "GITHUB_PAT",
    "github_pat",
    "GH_PAT",
    "gh_pat",
]


github_token = None
secret_label = None


for label in SECRET_ALIASES:

    try:
        value = secret_client.get_secret(
            label
        )

    except Exception:
        value = None

    if value:
        github_token = value.strip()
        secret_label = label
        break


if not github_token:
    raise RuntimeError(
        "No usable GitHub token found in Kaggle Secrets."
    )


print(
    f"[FOUND] {secret_label} "
    f"({len(github_token)} characters)"
)

print(
    "Token value will not be printed."
)


# =============================================================================
# 7. TEMPORARY GIT_ASKPASS
# =============================================================================

banner(
    "STAGE26-0C-GIT :: PUSH"
)


askpass_fd, askpass_path_str = tempfile.mkstemp(
    prefix="stage26_git_askpass_",
    suffix=".sh",
)

os.close(
    askpass_fd
)

askpass_path = Path(
    askpass_path_str
)


askpass_path.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' "x-access-token" ;;
  *Password*) printf '%s\\n' "$GITHUB_TOKEN" ;;
  *)          printf '%s\\n' "" ;;
esac
""",
    encoding="utf-8",
)


askpass_path.chmod(
    stat.S_IRUSR
    |
    stat.S_IWUSR
    |
    stat.S_IXUSR
)


push_env = os.environ.copy()

push_env[
    "GITHUB_TOKEN"
] = github_token

push_env[
    "GIT_ASKPASS"
] = str(
    askpass_path
)

push_env[
    "GIT_TERMINAL_PROMPT"
] = "0"


try:

    push = run(
        [
            "git",
            "push",
            "origin",
            "HEAD:main",
        ],
        env=push_env,
        check=True,
    )

    print(
        push.stdout
    )

finally:

    # Remove credential helper immediately.
    try:
        askpass_path.unlink()
    except FileNotFoundError:
        pass

    push_env.pop(
        "GITHUB_TOKEN",
        None,
    )

    github_token = None


# =============================================================================
# 8. FETCH REMOTE BACK
# =============================================================================

banner(
    "STAGE26-0C-GIT :: FETCH REMOTE VERIFICATION COPY"
)


fetch = git(
    "fetch",
    "origin",
    "main",
)

if fetch:
    print(fetch)


LOCAL_HEAD = git(
    "rev-parse",
    "HEAD",
)

REMOTE_HEAD = git(
    "rev-parse",
    "origin/main",
)


print("Local HEAD   :", LOCAL_HEAD)
print("origin/main  :", REMOTE_HEAD)


if LOCAL_HEAD != REMOTE_HEAD:
    raise RuntimeError(
        "Remote main does not match the Stage26 protocol commit."
    )


# =============================================================================
# 9. VERIFY REMOTE FILE BYTES
# =============================================================================

banner(
    "STAGE26-0C-GIT :: REMOTE ARTIFACT SHA VERIFICATION"
)


remote_checks = [
    (
        "measurement_protocol.json",
        PROTOCOL_REL,
        EXPECTED_PROTOCOL_SHA256,
    ),
    (
        "cpu_execution_plan.json",
        PLAN_REL,
        EXPECTED_PLAN_SHA256,
    ),
    (
        "stage26_0c_protocol_freeze_record.json",
        FREEZE_REL,
        EXPECTED_FREEZE_RECORD_SHA256,
    ),
    (
        "stage26_0c_package_manifest.json",
        PACKAGE_MANIFEST_REL,
        EXPECTED_PACKAGE_MANIFEST_SHA256,
    ),
]


for name, path, expected_sha in remote_checks:

    remote_bytes = git_blob_bytes(
        "origin/main",
        path,
    )

    remote_sha = sha256_bytes(
        remote_bytes
    )

    ok = (
        remote_sha
        ==
        expected_sha
    )

    print(
        f"{name:45s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{remote_sha}"
    )

    if not ok:
        raise RuntimeError(
            f"Remote SHA mismatch: {name}"
        )


# =============================================================================
# 10. VERIFY REMOTE PROTOCOL CONTENT
# =============================================================================

banner(
    "STAGE26-0C-GIT :: REMOTE PROTOCOL CONTENT GATE"
)


remote_protocol_bytes = git_blob_bytes(
    "origin/main",
    PROTOCOL_REL,
)

remote_protocol = json.loads(
    remote_protocol_bytes.decode(
        "utf-8"
    )
)


expected_status = (
    "FROZEN_BEFORE_FIRST_STAGE26_RESOURCE_MEASUREMENT"
)

actual_status = remote_protocol.get(
    "status"
)


print(
    "Remote protocol status:",
    actual_status,
)


if actual_status != expected_status:
    raise RuntimeError(
        "Unexpected remote protocol status."
    )


boundary = remote_protocol[
    "scientific_boundary_at_freeze"
]


print("\nRemote scientific boundary:")

for key, value in boundary.items():
    print(
        f"  {key:35s}: {value}"
    )


expected_boundary = {
    "models_deserialized": False,
    "models_loaded_for_stage26": False,
    "stage26_inference_calls": 0,
    "stage26_timing_observations": 0,
    "stage26_memory_measurements": 0,
    "stage26_extraction_measurements": 0,
    "holdout_reopened": False,
    "gpu_used": False,
}


for key, expected_value in expected_boundary.items():

    actual_value = boundary.get(
        key
    )

    if actual_value != expected_value:
        raise RuntimeError(
            f"Remote scientific boundary mismatch: "
            f"{key}={actual_value!r}, "
            f"expected {expected_value!r}"
        )


# =============================================================================
# 11. FINAL REPOSITORY AUDIT
# =============================================================================

banner(
    "STAGE26-0C-GIT :: FINAL REPOSITORY AUDIT"
)


FINAL_STATUS = git(
    "status",
    "--porcelain",
)


print(
    "Repository clean:",
    FINAL_STATUS == "",
)


if FINAL_STATUS:
    print(
        FINAL_STATUS
    )

    raise RuntimeError(
        "Repository is not clean after Stage26 protocol push."
    )


# Confirm commit exists remotely through ls-remote as an independent ref check.
remote_ls = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


remote_ls_sha = (
    remote_ls.split()[0]
    if remote_ls
    else None
)


print(
    "ls-remote main:",
    remote_ls_sha,
)


if remote_ls_sha != NEW_HEAD:
    raise RuntimeError(
        "git ls-remote does not confirm the pushed Stage26 commit."
    )


# =============================================================================
# 12. CLOSURE
# =============================================================================

banner(
    "STAGE26-0C-GIT COMPLETE"
)


print(
    "PARENT HEAD:"
)
print(
    " ",
    EXPECTED_PARENT_HEAD,
)

print(
    "\nSTAGE26 PROTOCOL COMMIT:"
)
print(
    " ",
    NEW_HEAD,
)

print(
    "\nREMOTE MAIN:"
)
print(
    " ",
    REMOTE_HEAD,
)

print(
    "\nMEASUREMENT PROTOCOL SHA256:"
)
print(
    " ",
    EXPECTED_PROTOCOL_SHA256,
)

print(
    "\nCPU EXECUTION PLAN SHA256:"
)
print(
    " ",
    EXPECTED_PLAN_SHA256,
)

print(
    "\nPACKAGE MANIFEST SHA256:"
)
print(
    " ",
    EXPECTED_PACKAGE_MANIFEST_SHA256,
)

print(
    "\nREMOTE VERIFICATION:"
)
print(
    "  COMMIT MATCH       : PASS"
)
print(
    "  PROTOCOL SHA       : PASS"
)
print(
    "  EXECUTION PLAN SHA : PASS"
)
print(
    "  FREEZE RECORD SHA  : PASS"
)
print(
    "  PACKAGE SHA        : PASS"
)
print(
    "  SCIENTIFIC BOUNDARY: PASS"
)
print(
    "  REPOSITORY CLEAN   : PASS"
)

print(
    "\nSCIENTIFIC STATE:"
)
print(
    "  STAGE26-0 PROTOCOL IS NOW DURABLY FROZEN ON GITHUB"
)
print(
    "  NO MODEL LOADED FOR STAGE26"
)
print(
    "  NO INFERENCE EXECUTED"
)
print(
    "  NO LATENCY OBSERVED"
)
print(
    "  NO MEMORY PROFILE OBSERVED"
)
print(
    "  NO EXTRACTION TIMING OBSERVED"
)
print(
    "  GPU NOT USED"
)

print(
    "\nNEXT:"
)
print(
    "  STAGE26-0D — CPU EXECUTION PREFLIGHT"
)
print(
    "  Model deserialization becomes permitted."
)
print(
    "  Timing remains prohibited."
)
print(
    "  We will verify each frozen model can reproduce the exact"
)
print(
    "  required forward/predict path under the locked CPU semantics."
)


STAGE26-0C-GIT :: PRE-COMMIT GATE
Expected parent HEAD : f3957b964e659392c57d1b12881163288f2efedc
Actual parent HEAD   : f3957b964e659392c57d1b12881163288f2efedc


RuntimeError: No uncommitted Stage26 protocol package found.

In [10]:
# =============================================================================
# STAGE26-0C-GIT-RECOVERY
#
# Safe/idempotent recovery after accidental double-run.
#
# This cell:
#   - inspects current local commit
#   - verifies frozen Stage26 protocol hashes
#   - checks whether the Stage26 protocol commit exists locally
#   - fetches origin/main
#   - pushes ONLY if local Stage26 commit is ahead of origin/main
#   - verifies exact remote file bytes/hashes
#   - confirms repository clean
#
# NO MODEL LOAD
# NO INFERENCE
# NO TIMING
# NO MEMORY PROFILING
# NO GPU
# =============================================================================

from __future__ import annotations

import os
import stat
import hashlib
import tempfile
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient


# =============================================================================
# CONFIG
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT_HEAD = (
    "f9307d5f7a4c8fd53886d66b3e5f8e74a65d87c6"
)

EXPECTED_COMMIT_MESSAGE = (
    "stage26: freeze deployment resource measurement protocol"
)

PROTOCOL_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/measurement_protocol.json"
)

PLAN_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/cpu_execution_plan.json"
)

FREEZE_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/stage26_0c_protocol_freeze_record.json"
)

PACKAGE_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/stage26_0c_package_manifest.json"
)


EXPECTED_HASHES = {
    PROTOCOL_REL:
        "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625",

    PLAN_REL:
        "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363",

    FREEZE_REL:
        "95c1cc93c377d8f017de24e67f6fc04e192babdce4a32244df9741b5465191dc",

    PACKAGE_REL:
        "620c4174db2df59ab178b7f11779ce442a4623df8faa34f9e4f0b0ccc484972e",
}


# =============================================================================
# HELPERS
# =============================================================================

def banner(text):
    print("\n" + "=" * 100)
    print(text)
    print("=" * 100)


def run(cmd, *, env=None, check=True):
    p = subprocess.run(
        cmd,
        cwd=REPO_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "$ " + " ".join(cmd) + "\n\n" + p.stdout
        )

    return p


def git(*args, env=None, check=True):
    return run(
        ["git", *args],
        env=env,
        check=check,
    ).stdout.strip()


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(8 * 1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def git_blob_bytes(ref, path):
    p = subprocess.run(
        [
            "git",
            "show",
            f"{ref}:{path}",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stderr.decode(
                "utf-8",
                errors="replace",
            )
        )

    return p.stdout


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


# =============================================================================
# 1. LOCAL STATE
# =============================================================================

banner("STAGE26-0C-GIT-RECOVERY :: LOCAL STATE")

LOCAL_HEAD = git(
    "rev-parse",
    "HEAD",
)

LOCAL_PARENT = git(
    "rev-parse",
    "HEAD^",
    check=False,
)

LOCAL_SUBJECT = git(
    "log",
    "-1",
    "--pretty=%s",
)

STATUS = git(
    "status",
    "--porcelain",
)


print("Current HEAD  :", LOCAL_HEAD)
print("Parent HEAD   :", LOCAL_PARENT)
print("Commit subject:", LOCAL_SUBJECT)
print("Repo clean    :", STATUS == "")

if STATUS:
    print("\nRepository status:")
    print(STATUS)


# =============================================================================
# 2. IDENTIFY WHETHER FIRST RUN COMMITTED STAGE26
# =============================================================================

banner("STAGE26-0C-GIT-RECOVERY :: COMMIT IDENTITY")

stage26_commit_present = (
    LOCAL_PARENT == EXPECTED_PARENT_HEAD
    and
    LOCAL_SUBJECT == EXPECTED_COMMIT_MESSAGE
)


print(
    "Expected parent       :",
    EXPECTED_PARENT_HEAD,
)

print(
    "Stage26 commit present:",
    stage26_commit_present,
)


if not stage26_commit_present:

    raise RuntimeError(
        "\nThe current HEAD is not the expected Stage26 protocol commit.\n"
        "Do NOT run profiling.\n"
        "Paste this cell's output so we can inspect the exact Git state."
    )


# =============================================================================
# 3. VERIFY LOCAL COMMITTED FILE HASHES
# =============================================================================

banner("STAGE26-0C-GIT-RECOVERY :: LOCAL HASH GATE")


for relpath, expected_sha in EXPECTED_HASHES.items():

    disk_path = (
        REPO_DIR
        / relpath
    )

    if not disk_path.exists():
        raise FileNotFoundError(
            disk_path
        )

    actual_disk_sha = sha256_file(
        disk_path
    )

    committed_bytes = git_blob_bytes(
        "HEAD",
        relpath,
    )

    committed_sha = sha256_bytes(
        committed_bytes
    )

    disk_ok = (
        actual_disk_sha
        ==
        expected_sha
    )

    commit_ok = (
        committed_sha
        ==
        expected_sha
    )

    print(
        f"{Path(relpath).name:45s} "
        f"disk={'PASS' if disk_ok else 'FAIL'} "
        f"commit={'PASS' if commit_ok else 'FAIL'}"
    )

    if not disk_ok or not commit_ok:
        raise RuntimeError(
            f"Stage26 frozen hash mismatch: {relpath}"
        )


# =============================================================================
# 4. FETCH REMOTE
# =============================================================================

banner("STAGE26-0C-GIT-RECOVERY :: FETCH REMOTE")

fetch = git(
    "fetch",
    "origin",
    "main",
    check=False,
)

if fetch:
    print(fetch)


REMOTE_HEAD = git(
    "rev-parse",
    "origin/main",
)


print("Local HEAD :", LOCAL_HEAD)
print("Remote HEAD:", REMOTE_HEAD)


# =============================================================================
# 5. DETERMINE RELATIONSHIP
# =============================================================================

banner("STAGE26-0C-GIT-RECOVERY :: LOCAL / REMOTE RELATIONSHIP")


if REMOTE_HEAD == LOCAL_HEAD:

    relation = "REMOTE_ALREADY_VERIFIED_COMMIT"

elif REMOTE_HEAD == EXPECTED_PARENT_HEAD:

    relation = "LOCAL_STAGE26_COMMIT_AHEAD_BY_ONE"

else:

    relation = "UNEXPECTED_DIVERGENCE"


print("Relationship:", relation)


if relation == "UNEXPECTED_DIVERGENCE":

    raise RuntimeError(
        "\norigin/main is neither the Stage26 parent nor the local "
        "Stage26 protocol commit.\n"
        "Do NOT push automatically.\n"
        "Paste this output so we can inspect the divergence."
    )


# =============================================================================
# 6. PUSH ONLY IF REQUIRED
# =============================================================================

if relation == "LOCAL_STAGE26_COMMIT_AHEAD_BY_ONE":

    banner("STAGE26-0C-GIT-RECOVERY :: PUSH REQUIRED")

    secret_client = UserSecretsClient()

    aliases = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "gh_token",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
        "gh_pat",
    ]

    token = None
    token_label = None

    for label in aliases:

        try:
            value = secret_client.get_secret(
                label
            )

        except Exception:
            value = None

        if value:
            token = value.strip()
            token_label = label
            break


    if not token:
        raise RuntimeError(
            "GitHub token unavailable from Kaggle Secrets."
        )


    print(
        f"[FOUND] {token_label} "
        f"({len(token)} characters)"
    )

    print(
        "Token value will not be printed."
    )


    fd, askpass_name = tempfile.mkstemp(
        prefix="stage26_askpass_",
        suffix=".sh",
    )

    os.close(fd)

    askpass = Path(
        askpass_name
    )

    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' "x-access-token" ;;
  *Password*) printf '%s\\n' "$GITHUB_TOKEN" ;;
  *)          printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )

    askpass.chmod(
        stat.S_IRUSR
        |
        stat.S_IWUSR
        |
        stat.S_IXUSR
    )


    env = os.environ.copy()

    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"


    try:

        result = run(
            [
                "git",
                "push",
                "origin",
                "HEAD:main",
            ],
            env=env,
        )

        print(result.stdout)

    finally:

        try:
            askpass.unlink()
        except FileNotFoundError:
            pass

        token = None


    # Fetch exact remote state again.
    git(
        "fetch",
        "origin",
        "main",
    )

    REMOTE_HEAD = git(
        "rev-parse",
        "origin/main",
    )


# =============================================================================
# 7. REMOTE COMMIT GATE
# =============================================================================

banner("STAGE26-0C-GIT-RECOVERY :: REMOTE COMMIT GATE")

print("Local HEAD :", LOCAL_HEAD)
print("Remote HEAD:", REMOTE_HEAD)


if REMOTE_HEAD != LOCAL_HEAD:
    raise RuntimeError(
        "Remote Stage26 protocol commit is not identical to local HEAD."
    )


ls_remote = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

LS_REMOTE_SHA = (
    ls_remote.split()[0]
    if ls_remote
    else None
)


print("ls-remote :", LS_REMOTE_SHA)


if LS_REMOTE_SHA != LOCAL_HEAD:
    raise RuntimeError(
        "Independent ls-remote verification failed."
    )


# =============================================================================
# 8. REMOTE FILE HASH GATE
# =============================================================================

banner("STAGE26-0C-GIT-RECOVERY :: REMOTE FILE HASH GATE")


for relpath, expected_sha in EXPECTED_HASHES.items():

    remote_bytes = git_blob_bytes(
        "origin/main",
        relpath,
    )

    remote_sha = sha256_bytes(
        remote_bytes
    )

    ok = (
        remote_sha
        ==
        expected_sha
    )

    print(
        f"{Path(relpath).name:45s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{remote_sha}"
    )

    if not ok:
        raise RuntimeError(
            f"Remote frozen hash mismatch: {relpath}"
        )


# =============================================================================
# 9. FINAL CLEANLINESS
# =============================================================================

banner("STAGE26-0C-GIT-RECOVERY :: FINAL AUDIT")


FINAL_STATUS = git(
    "status",
    "--porcelain",
)


print(
    "Repository clean:",
    FINAL_STATUS == "",
)


if FINAL_STATUS:
    print(FINAL_STATUS)

    raise RuntimeError(
        "Repository is dirty after recovery."
    )


# =============================================================================
# 10. CLOSURE
# =============================================================================

banner("STAGE26-0C-GIT RECOVERY COMPLETE")


print("STAGE26 PROTOCOL COMMIT:")
print(" ", LOCAL_HEAD)

print("\norigin/main:")
print(" ", REMOTE_HEAD)

print("\nProtocol SHA256:")
print(
    " ",
    EXPECTED_HASHES[
        PROTOCOL_REL
    ],
)

print("\nExecution-plan SHA256:")
print(
    " ",
    EXPECTED_HASHES[
        PLAN_REL
    ],
)

print(
    "\nREMOTE VERIFICATION:"
)
print(
    "  COMMIT IDENTITY     : PASS"
)
print(
    "  LOCAL FILE HASHES   : PASS"
)
print(
    "  REMOTE COMMIT       : PASS"
)
print(
    "  REMOTE FILE HASHES  : PASS"
)
print(
    "  LS-REMOTE CHECK     : PASS"
)
print(
    "  REPOSITORY CLEAN    : PASS"
)

print(
    "\nSCIENTIFIC STATE:"
)
print(
    "  STAGE26-0 PROTOCOL DURABLY FROZEN"
)
print(
    "  NO MODEL LOAD CAUSED BY THIS RECOVERY"
)
print(
    "  NO INFERENCE"
)
print(
    "  NO LATENCY MEASUREMENT"
)
print(
    "  NO MEMORY MEASUREMENT"
)
print(
    "  GPU NOT USED"
)

print(
    "\nNEXT:"
)
print(
    "  STAGE26-0D — CPU EXECUTION PREFLIGHT"
)


STAGE26-0C-GIT-RECOVERY :: LOCAL STATE
Current HEAD  : f3957b964e659392c57d1b12881163288f2efedc
Parent HEAD   : f9307d5f7a4c8fd53886d66b3e5f8e74a65d87c6
Commit subject: stage26: freeze deployment resource measurement protocol
Repo clean    : True

STAGE26-0C-GIT-RECOVERY :: COMMIT IDENTITY
Expected parent       : f9307d5f7a4c8fd53886d66b3e5f8e74a65d87c6
Stage26 commit present: True

STAGE26-0C-GIT-RECOVERY :: LOCAL HASH GATE
measurement_protocol.json                     disk=PASS commit=PASS
cpu_execution_plan.json                       disk=PASS commit=PASS
stage26_0c_protocol_freeze_record.json        disk=PASS commit=PASS
stage26_0c_package_manifest.json              disk=PASS commit=PASS

STAGE26-0C-GIT-RECOVERY :: FETCH REMOTE
From https://github.com/themubasshir/ids2018-validation-safe-ablation
 * branch            main       -> FETCH_HEAD
Local HEAD : f3957b964e659392c57d1b12881163288f2efedc
Remote HEAD: f3957b964e659392c57d1b12881163288f2efedc

STAGE26-0C-GIT-RECOVERY :: LOCAL

In [11]:
# =============================================================================
# STAGE26-0D — CPU EXECUTION PREFLIGHT
#
# PURPOSE
# -------
# Validate that every frozen Stage26 target can be deserialized and can execute
# its exact CPU inference path under BOTH frozen CPU execution modes.
#
# SCIENTIFIC BOUNDARY
# -------------------
# ALLOWED:
#   - model deserialization
#   - deterministic synthetic input construction
#   - exactly one untimed prediction per target × CPU mode
#   - output shape / finiteness validation
#   - architecture parameter-count validation
#   - prediction fingerprinting (SHA256 only; values not inspected)
#
# FORBIDDEN:
#   - perf_counter / monotonic / time.time / CUDA events
#   - latency measurement
#   - throughput measurement
#   - memory profiling
#   - warmup loops
#   - timed loops
#   - holdout access
#   - model selection / threshold tuning
#   - GPU
#
# TARGETS:
#   8 deployment targets × 2 CPU modes = 16 preflight executions.
#
# NOTE:
#   The preflight uses fresh subprocesses so thread environment variables and
#   CPU affinity are applied before ML/numerical framework import.
# =============================================================================

from __future__ import annotations

import os
import sys
import json
import stat
import shutil
import hashlib
import subprocess
import textwrap
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. PATHS / FROZEN IDENTITIES
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

WORKERS_DIR = (
    STAGE26_ROOT
    / "workers"
)

RUNTIME_DIR = (
    STAGE26_ROOT
    / "runtime"
)

PROTOCOL_DIR = (
    STAGE26_ROOT
    / "protocol"
)

PREFLIGHT_LOCAL_DIR = (
    STAGE26_ROOT
    / "preflight"
)

PREFLIGHT_REPO_DIR = (
    REPO_DIR
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0d_cpu_preflight"
)

for d in [
    WORKERS_DIR,
    RUNTIME_DIR,
    PROTOCOL_DIR,
    PREFLIGHT_LOCAL_DIR,
]:
    d.mkdir(
        parents=True,
        exist_ok=True,
    )


EXPECTED_HEAD = (
    "f3957b964e659392c57d1b12881163288f2efedc"
)

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

EXPECTED_PLAN_SHA256 = (
    "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363"
)

MEASUREMENT_SEED = 26042


# =============================================================================
# 1. HELPERS — NO TIMING FUNCTIONS
# =============================================================================

def banner(text: str):
    print("\n" + "=" * 104)
    print(text)
    print("=" * 104)


def git(*args):
    p = subprocess.run(
        ["git", *args],
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(path: Path):
    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def write_json(path: Path, payload):
    path.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )


# =============================================================================
# 2. PRE-PREFLIGHT SCIENTIFIC GATE
# =============================================================================

banner(
    "STAGE26-0D :: SCIENTIFIC / GIT GATE"
)

HEAD = git(
    "rev-parse",
    "HEAD",
)

REMOTE = git(
    "rev-parse",
    "origin/main",
)

STATUS = git(
    "status",
    "--porcelain",
)

print("Expected HEAD :", EXPECTED_HEAD)
print("Local HEAD    :", HEAD)
print("origin/main   :", REMOTE)
print("Repo clean    :", STATUS == "")


if HEAD != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected local HEAD."
    )

if REMOTE != EXPECTED_HEAD:
    raise RuntimeError(
        "origin/main no longer matches the frozen Stage26 protocol commit."
    )

if STATUS:
    raise RuntimeError(
        "Repository must be clean before CPU preflight:\n"
        + STATUS
    )


MEASUREMENT_PROTOCOL = (
    REPO_DIR
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXECUTION_PLAN = (
    REPO_DIR
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "cpu_execution_plan.json"
)


protocol_sha = sha256_file(
    MEASUREMENT_PROTOCOL
)

plan_sha = sha256_file(
    EXECUTION_PLAN
)


print(
    "\nmeasurement_protocol SHA:",
    protocol_sha,
)

print(
    "execution_plan SHA      :",
    plan_sha,
)


if protocol_sha != EXPECTED_PROTOCOL_SHA256:
    raise RuntimeError(
        "Frozen measurement protocol SHA mismatch."
    )

if plan_sha != EXPECTED_PLAN_SHA256:
    raise RuntimeError(
        "Frozen execution-plan SHA mismatch."
    )


# =============================================================================
# 3. FROZEN SOURCE / ARTIFACT PATHS
# =============================================================================

banner(
    "STAGE26-0D :: SOURCE / ARTIFACT GATE"
)


PATHS = {
    # -------------------------------------------------------------------------
    # Classical models
    # -------------------------------------------------------------------------
    "xgboost": (
        REPO_DIR
        / "results/stage16_classical_benchmark_checkpoint/"
          "stage16_3_tuned_models/XGBOOST_tuned.joblib"
    ),

    "lightgbm": (
        REPO_DIR
        / "results/stage16_classical_benchmark_checkpoint/"
          "stage16_3_tuned_models/LIGHTGBM_tuned.joblib"
    ),

    "catboost": (
        REPO_DIR
        / "results/stage16_classical_benchmark_checkpoint/"
          "stage16_3_tuned_models/CATBOOST_tuned.joblib"
    ),

    # -------------------------------------------------------------------------
    # Stage15 FT checkpoints
    # -------------------------------------------------------------------------
    "ft_seed7": (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_4b_models/FT_BALANCED_seed_7_best_extended.pt"
    ),

    "ft_seed29": (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_4a_models/FT_BALANCED_seed_29_best.pt"
    ),

    "ft_seed101": (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_4a_models/FT_BALANCED_seed_101_best.pt"
    ),

    "ft_seed313": (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_4c_models/FT_BALANCED_seed_313_best.pt"
    ),

    "ft_seed997": (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_4c_models/FT_BALANCED_seed_997_best.pt"
    ),

    "ft_module": (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "ft_transformer_numeric.py"
    ),

    "ft_architecture": (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_4c_frozen_architecture.json"
    ),

    "scaler": (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_2_standard_scaler.joblib"
    ),

    # -------------------------------------------------------------------------
    # Packet models
    # -------------------------------------------------------------------------
    "cnn_state": (
        REPO_DIR
        / "results/stage20_1e_training/"
          "stage20_1e2_epoch10_model_state_dict.pt"
    ),

    "cnn_module": (
        REPO_DIR
        / "scripts/stage20_masked_cnn.py"
    ),

    "vit_state": (
        REPO_DIR
        / "results/stage21_architecture/"
          "stage21_2_epoch10_model_state_dict.pt"
    ),

    "vit_module": (
        REPO_DIR
        / "scripts/stage21_masked_vit.py"
    ),

    "packet_encoder": (
        REPO_DIR
        / "scripts/stage20_packet_image_encoder.py"
    ),
}


for name, path in PATHS.items():

    if not path.exists():
        raise FileNotFoundError(
            f"{name}: {path}"
        )

    print(
        f"[OK] {name:18s} "
        f"{path.relative_to(REPO_DIR)}"
    )


# =============================================================================
# 4. SOURCE-CODE SHA GATES
# =============================================================================

banner(
    "STAGE26-0D :: EXECUTABLE SOURCE SHA GATE"
)


SOURCE_HASHES = {
    "ft_module": (
        "8233a7b62e7045d7c920c15fd8ec974ed8b3db6f2f131dc4c64ba0ce0133dcfc"
    ),

    "cnn_module": (
        "3638ae622017a36e6eeb33f227135829695ff2f3581c9b43787a02c1a440b9d4"
    ),

    "vit_module": (
        "3af99e4ea7061c68a676dc8fa7e485a7d13278f8947e4f8a8fbf2069dc31e3cb"
    ),

    "packet_encoder": (
        "9883fe2b27020aaff707a753123b35eb3223d21abf295d056ec233e532f94222"
    ),

    "scaler": (
        "ac1e3a9b0a2409edcd98c293f31c22b9c140cc882518881565d16eb1e2a573b5"
    ),
}


for name, expected in SOURCE_HASHES.items():

    actual = sha256_file(
        PATHS[name]
    )

    ok = (
        actual
        ==
        expected
    )

    print(
        f"{name:18s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual}"
    )

    if not ok:
        raise RuntimeError(
            f"Source/dependency SHA mismatch: {name}"
        )


# =============================================================================
# 5. FROZEN CPU MODES
# =============================================================================

CPU_MODES = {
    "CPU_1_PHYSICAL_CORE": {
        "threads": 1,
        "affinity": [0],
    },

    "CPU_2_PHYSICAL_CORE": {
        "threads": 2,
        "affinity": [0, 1],
    },
}


TARGETS = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
    "ENS_LGBM_XGB_EQUAL",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
]


print(
    "\nPreflight targets:",
    len(TARGETS),
)

print(
    "CPU modes:",
    len(CPU_MODES),
)

print(
    "Total subprocess checks:",
    len(TARGETS) * len(CPU_MODES),
)


# =============================================================================
# 6. CREATE PREFLIGHT WORKER
#
# IMPORTANT:
#   This worker intentionally contains NO performance clock.
# =============================================================================

banner(
    "STAGE26-0D :: CREATE UNTIMERED PREFLIGHT WORKER"
)


WORKER_PATH = (
    WORKERS_DIR
    / "stage26_cpu_preflight_worker.py"
)


worker_source = r'''
from __future__ import annotations

import os
import sys
import json
import hashlib
import importlib.util
from pathlib import Path


# =============================================================================
# No performance timing is permitted in this worker.
# =============================================================================

FORBIDDEN_CLOCK_NAMES = [
    "perf_counter",
    "monotonic",
    "process_time",
    "thread_time",
    "time_ns",
]


def sha256_array(array):
    import numpy as np

    a = np.ascontiguousarray(
        array
    )

    h = hashlib.sha256()

    h.update(
        str(
            a.dtype
        ).encode("ascii")
    )

    h.update(
        json.dumps(
            list(
                a.shape
            )
        ).encode("ascii")
    )

    h.update(
        a.tobytes(
            order="C"
        )
    )

    return h.hexdigest()


def import_path(name, path):

    spec = importlib.util.spec_from_file_location(
        name,
        str(path),
    )

    if spec is None or spec.loader is None:
        raise RuntimeError(
            f"Unable to import {path}"
        )

    module = importlib.util.module_from_spec(
        spec
    )

    spec.loader.exec_module(
        module
    )

    return module


def extract_torch_state(obj):

    import torch

    if not isinstance(
        obj,
        dict,
    ):
        raise TypeError(
            f"Checkpoint root type unsupported: {type(obj)}"
        )

    # Direct state_dict.
    if obj and all(
        torch.is_tensor(v)
        for v in obj.values()
    ):
        return (
            obj,
            "DIRECT_STATE_DICT",
        )

    for key in [
        "model_state_dict",
        "state_dict",
        "model_state",
        "network_state_dict",
        "model",
    ]:

        candidate = obj.get(
            key
        )

        if (
            isinstance(
                candidate,
                dict,
            )
            and
            candidate
            and
            all(
                torch.is_tensor(v)
                for v in candidate.values()
            )
        ):
            return (
                candidate,
                key,
            )

    raise RuntimeError(
        "Unable to identify torch state_dict layout. "
        f"Top-level keys={list(obj.keys())[:30]}"
    )


def torch_load_state(path):

    import torch

    try:
        obj = torch.load(
            str(path),
            map_location="cpu",
            weights_only=True,
        )

        load_mode = (
            "torch.load(weights_only=True)"
        )

    except TypeError:

        obj = torch.load(
            str(path),
            map_location="cpu",
        )

        load_mode = (
            "torch.load(default)"
        )

    state, state_layout = (
        extract_torch_state(
            obj
        )
    )

    return (
        state,
        load_mode,
        state_layout,
    )


def configure_torch_threads(thread_count):

    import torch

    torch.set_num_threads(
        int(
            thread_count
        )
    )

    try:
        torch.set_num_interop_threads(
            1
        )
    except RuntimeError:
        # This is still checked below. A mismatch fails the preflight.
        pass

    return {
        "torch_num_threads":
            int(
                torch.get_num_threads()
            ),

        "torch_num_interop_threads":
            int(
                torch.get_num_interop_threads()
            ),
    }


def build_group_a_input(
    *,
    repo,
    batch_size,
    seed,
):

    import joblib
    import numpy as np

    scaler_path = (
        repo
        / "results/stage15_transformer_checkpoint/"
          "stage15_2_standard_scaler.joblib"
    )

    scaler = joblib.load(
        scaler_path
    )

    if int(
        scaler.n_features_in_
    ) != 70:
        raise RuntimeError(
            "Frozen scaler does not have 70 features."
        )

    # Frozen implementation of:
    # "deterministic from frozen seed derived from batch size"
    #
    # No performance result is consulted.
    derived_seed = (
        int(seed)
        +
        int(batch_size)
    )

    rng = np.random.default_rng(
        derived_seed
    )

    Z = rng.normal(
        loc=0.0,
        scale=1.0,
        size=(
            int(batch_size),
            70,
        ),
    ).astype(
        np.float32
    )

    mean = np.asarray(
        scaler.mean_,
        dtype=np.float32,
    )

    scale = np.asarray(
        scaler.scale_,
        dtype=np.float32,
    )

    X_raw = (
        mean[None, :]
        +
        Z
        *
        scale[None, :]
    ).astype(
        np.float32,
        copy=False,
    )

    if not np.isfinite(
        Z
    ).all():
        raise RuntimeError(
            "Non-finite FT synthetic input."
        )

    if not np.isfinite(
        X_raw
    ).all():
        raise RuntimeError(
            "Non-finite tree synthetic input."
        )

    return {
        "Z": Z,
        "X_raw": X_raw,
        "derived_seed": derived_seed,
    }


def make_ipv4_packet(
    rng,
    protocol,
    length,
):

    import numpy as np

    if length < 40:
        raise ValueError(
            "Synthetic packet length must be >=40."
        )

    packet = bytearray(
        rng.integers(
            0,
            256,
            size=int(length),
            dtype=np.uint8,
        ).tobytes()
    )

    # IPv4 + IHL 5.
    packet[0] = 0x45

    # Total length.
    packet[2:4] = int(
        length
    ).to_bytes(
        2,
        "big",
    )

    # Fragment offset = zero.
    packet[6] = (
        packet[6]
        &
        0xE0
    )

    packet[7] = 0

    # Protocol 6 TCP / 17 UDP.
    packet[9] = int(
        protocol
    )

    return bytes(
        packet
    )


def build_packet_input(
    *,
    repo,
    batch_size,
    seed,
):

    import numpy as np

    encoder = import_path(
        "stage26_packet_encoder",
        (
            repo
            / "scripts/stage20_packet_image_encoder.py"
        ),
    )

    derived_seed = (
        int(seed)
        +
        1_000_000
        +
        int(batch_size)
    )

    rng = np.random.default_rng(
        derived_seed
    )

    images_uint8 = np.zeros(
        (
            int(batch_size),
            1,
            64,
            256,
        ),
        dtype=np.uint8,
    )

    masks = np.zeros(
        (
            int(batch_size),
            1,
            64,
            256,
        ),
        dtype=np.bool_,
    )

    for flow_idx in range(
        int(batch_size)
    ):

        packet_count = int(
            rng.integers(
                1,
                65,
            )
        )

        packets = []

        for _ in range(
            packet_count
        ):

            protocol = (
                6
                if int(
                    rng.integers(
                        0,
                        2,
                    )
                ) == 0
                else 17
            )

            packet_length = int(
                rng.integers(
                    40,
                    257,
                )
            )

            packets.append(
                make_ipv4_packet(
                    rng,
                    protocol,
                    packet_length,
                )
            )

        image, mask = (
            encoder.encode_flow(
                packets
            )
        )

        images_uint8[
            flow_idx,
            0,
        ] = image

        masks[
            flow_idx,
            0,
        ] = mask

    images_scaled = (
        images_uint8.astype(
            np.float32
        )
        /
        np.float32(
            255.0
        )
    )

    return {
        "image_uint8":
            images_uint8,

        "image_scaled":
            images_scaled,

        "padding_mask":
            masks,

        "derived_seed":
            derived_seed,
    }


def validate_probability_vector(
    probabilities,
    *,
    expected_rows,
):

    import numpy as np

    p = np.asarray(
        probabilities
    )

    p = p.reshape(
        -1
    )

    if p.shape != (
        int(expected_rows),
    ):
        raise RuntimeError(
            f"Unexpected probability shape: {p.shape}"
        )

    if not np.isfinite(
        p
    ).all():
        raise RuntimeError(
            "Non-finite prediction encountered."
        )

    if (
        (p < 0.0).any()
        or
        (p > 1.0).any()
    ):
        raise RuntimeError(
            "Probability outside [0,1]."
        )

    return p


def load_ft_model(
    *,
    repo,
    checkpoint_path,
):

    import torch

    ft_module = import_path(
        "stage26_ft_transformer",
        (
            repo
            / "results/stage15_transformer_checkpoint/"
              "ft_transformer_numeric.py"
        ),
    )

    arch_path = (
        repo
        / "results/stage15_transformer_checkpoint/"
          "stage15_4c_frozen_architecture.json"
    )

    arch_record = json.loads(
        arch_path.read_text(
            encoding="utf-8"
        )
    )

    arch = arch_record[
        "architecture"
    ]

    model = ft_module.NumericFTTransformer(
        n_features=int(
            arch_record[
                "input_predictor_count"
            ]
        ),
        d_token=int(
            arch["d_token"]
        ),
        n_heads=int(
            arch["n_heads"]
        ),
        n_layers=int(
            arch["n_layers"]
        ),
        d_ff=int(
            arch["d_ff"]
        ),
        dropout=float(
            arch["dropout"]
        ),
    )

    state, load_mode, layout = (
        torch_load_state(
            checkpoint_path
        )
    )

    model.load_state_dict(
        state,
        strict=True,
    )

    model.eval()

    parameter_count = int(
        sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        )
    )

    if parameter_count != 159169:
        raise RuntimeError(
            "FT parameter count mismatch: "
            f"{parameter_count}"
        )

    return (
        model,
        parameter_count,
        load_mode,
        layout,
    )


def main():

    config_path = Path(
        sys.argv[1]
    )

    config = json.loads(
        config_path.read_text(
            encoding="utf-8"
        )
    )

    repo = Path(
        config["repo"]
    )

    target = config[
        "target_id"
    ]

    hardware_mode = config[
        "hardware_mode"
    ]

    thread_count = int(
        config["thread_count"]
    )

    affinity = [
        int(x)
        for x in config[
            "affinity"
        ]
    ]

    batch_size = int(
        config.get(
            "batch_size",
            1,
        )
    )

    seed = int(
        config["measurement_seed"]
    )


    # -------------------------------------------------------------------------
    # Freeze CPU affinity BEFORE framework import.
    # -------------------------------------------------------------------------

    if hasattr(
        os,
        "sched_setaffinity",
    ):
        os.sched_setaffinity(
            0,
            set(
                affinity
            ),
        )

    observed_affinity = (
        sorted(
            os.sched_getaffinity(
                0
            )
        )
        if hasattr(
            os,
            "sched_getaffinity",
        )
        else None
    )


    result = {
        "status": "STARTED",
        "target_id": target,
        "hardware_mode": hardware_mode,
        "thread_count_requested": thread_count,
        "affinity_requested": affinity,
        "affinity_observed": observed_affinity,
        "batch_size": batch_size,
        "timing_performed": False,
        "memory_profiling_performed": False,
        "holdout_accessed": False,
        "gpu_used": False,
    }


    if (
        observed_affinity is not None
        and
        observed_affinity != affinity
    ):
        raise RuntimeError(
            f"Affinity mismatch: {observed_affinity} != {affinity}"
        )


    # -------------------------------------------------------------------------
    # Group A classical models
    # -------------------------------------------------------------------------

    if target in {
        "STAGE16_XGBOOST_TUNED",
        "STAGE16_LIGHTGBM_TUNED",
        "STAGE16_CATBOOST_TUNED",
        "ENS_LGBM_XGB_EQUAL",
    }:

        import joblib
        import numpy as np

        inputs = build_group_a_input(
            repo=repo,
            batch_size=batch_size,
            seed=seed,
        )

        X = inputs[
            "X_raw"
        ]

        result[
            "derived_input_seed"
        ] = inputs[
            "derived_seed"
        ]

        result[
            "input_shape"
        ] = list(
            X.shape
        )

        result[
            "input_dtype"
        ] = str(
            X.dtype
        )


        if target == "STAGE16_XGBOOST_TUNED":

            import xgboost

            path = (
                repo
                / "results/stage16_classical_benchmark_checkpoint/"
                  "stage16_3_tuned_models/XGBOOST_tuned.joblib"
            )

            model = joblib.load(
                path
            )

            if hasattr(
                model,
                "set_params",
            ):
                model.set_params(
                    n_jobs=thread_count
                )

            probabilities = (
                model.predict_proba(
                    X
                )[:, 1]
            )

            result[
                "framework_version"
            ] = xgboost.__version__

            result[
                "model_class"
            ] = (
                model.__class__.__module__
                + "."
                + model.__class__.__name__
            )


        elif target == "STAGE16_LIGHTGBM_TUNED":

            import lightgbm

            path = (
                repo
                / "results/stage16_classical_benchmark_checkpoint/"
                  "stage16_3_tuned_models/LIGHTGBM_tuned.joblib"
            )

            model = joblib.load(
                path
            )

            if hasattr(
                model,
                "set_params",
            ):
                model.set_params(
                    n_jobs=thread_count
                )

            probabilities = (
                model.predict_proba(
                    X
                )[:, 1]
            )

            result[
                "framework_version"
            ] = lightgbm.__version__

            result[
                "model_class"
            ] = (
                model.__class__.__module__
                + "."
                + model.__class__.__name__
            )


        elif target == "STAGE16_CATBOOST_TUNED":

            import catboost

            path = (
                repo
                / "results/stage16_classical_benchmark_checkpoint/"
                  "stage16_3_tuned_models/CATBOOST_tuned.joblib"
            )

            model = joblib.load(
                path
            )

            probabilities = (
                model.predict_proba(
                    X,
                    thread_count=thread_count,
                )[:, 1]
            )

            result[
                "framework_version"
            ] = catboost.__version__

            result[
                "model_class"
            ] = (
                model.__class__.__module__
                + "."
                + model.__class__.__name__
            )


        else:

            import xgboost
            import lightgbm

            xgb_path = (
                repo
                / "results/stage16_classical_benchmark_checkpoint/"
                  "stage16_3_tuned_models/XGBOOST_tuned.joblib"
            )

            lgb_path = (
                repo
                / "results/stage16_classical_benchmark_checkpoint/"
                  "stage16_3_tuned_models/LIGHTGBM_tuned.joblib"
            )

            xgb_model = joblib.load(
                xgb_path
            )

            lgb_model = joblib.load(
                lgb_path
            )

            if hasattr(
                xgb_model,
                "set_params",
            ):
                xgb_model.set_params(
                    n_jobs=thread_count
                )

            if hasattr(
                lgb_model,
                "set_params",
            ):
                lgb_model.set_params(
                    n_jobs=thread_count
                )

            xgb_p = (
                xgb_model.predict_proba(
                    X
                )[:, 1]
            )

            lgb_p = (
                lgb_model.predict_proba(
                    X
                )[:, 1]
            )

            probabilities = (
                np.asarray(
                    xgb_p,
                    dtype=np.float64,
                )
                +
                np.asarray(
                    lgb_p,
                    dtype=np.float64,
                )
            ) / 2.0

            result[
                "framework_version"
            ] = {
                "xgboost":
                    xgboost.__version__,
                "lightgbm":
                    lightgbm.__version__,
            }

            result[
                "model_class"
            ] = (
                "EqualWeight("
                + xgb_model.__class__.__name__
                + ","
                + lgb_model.__class__.__name__
                + ")"
            )


        p = validate_probability_vector(
            probabilities,
            expected_rows=batch_size,
        )

        result[
            "output_shape"
        ] = list(
            p.shape
        )

        result[
            "output_dtype"
        ] = str(
            p.dtype
        )

        result[
            "output_sha256"
        ] = sha256_array(
            p
        )


    # -------------------------------------------------------------------------
    # FT models
    # -------------------------------------------------------------------------

    elif target in {
        "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
        "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
    }:

        import numpy as np
        import torch

        thread_receipt = (
            configure_torch_threads(
                thread_count
            )
        )

        result.update(
            thread_receipt
        )

        inputs = build_group_a_input(
            repo=repo,
            batch_size=batch_size,
            seed=seed,
        )

        Z = inputs[
            "Z"
        ]

        result[
            "derived_input_seed"
        ] = inputs[
            "derived_seed"
        ]

        result[
            "input_shape"
        ] = list(
            Z.shape
        )

        result[
            "input_dtype"
        ] = str(
            Z.dtype
        )

        x = torch.from_numpy(
            Z
        )

        seed_paths = {
            7:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4b_models/FT_BALANCED_seed_7_best_extended.pt",

            29:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4a_models/FT_BALANCED_seed_29_best.pt",

            101:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4a_models/FT_BALANCED_seed_101_best.pt",

            313:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4c_models/FT_BALANCED_seed_313_best.pt",

            997:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4c_models/FT_BALANCED_seed_997_best.pt",
        }

        seeds = (
            [7, 29, 101, 313, 997]
            if target
            ==
            "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING"
            else
            [7]
        )

        member_probabilities = []

        load_receipts = []

        for checkpoint_seed in seeds:

            (
                model,
                parameter_count,
                load_mode,
                layout,
            ) = load_ft_model(
                repo=repo,
                checkpoint_path=seed_paths[
                    checkpoint_seed
                ],
            )

            with torch.inference_mode():

                logits = model(
                    x
                )

                member_p = torch.sigmoid(
                    logits
                ).detach().cpu().numpy()

            member_probabilities.append(
                member_p
            )

            load_receipts.append(
                {
                    "seed":
                        checkpoint_seed,

                    "parameter_count":
                        parameter_count,

                    "load_mode":
                        load_mode,

                    "state_layout":
                        layout,
                }
            )

        probabilities = np.mean(
            np.stack(
                member_probabilities,
                axis=0,
            ),
            axis=0,
        )

        p = validate_probability_vector(
            probabilities,
            expected_rows=batch_size,
        )

        result[
            "framework_version"
        ] = torch.__version__

        result[
            "model_class"
        ] = (
            "NumericFTTransformer"
        )

        result[
            "checkpoint_count"
        ] = len(
            seeds
        )

        result[
            "load_receipts"
        ] = load_receipts

        result[
            "output_shape"
        ] = list(
            p.shape
        )

        result[
            "output_dtype"
        ] = str(
            p.dtype
        )

        result[
            "output_sha256"
        ] = sha256_array(
            p
        )


    # -------------------------------------------------------------------------
    # CNN / ViT
    # -------------------------------------------------------------------------

    elif target in {
        "STAGE20_MASKED_CNN_V1",
        "STAGE21_MASKED_VIT_V1",
    }:

        import numpy as np
        import torch

        thread_receipt = (
            configure_torch_threads(
                thread_count
            )
        )

        result.update(
            thread_receipt
        )

        inputs = build_packet_input(
            repo=repo,
            batch_size=batch_size,
            seed=seed,
        )

        result[
            "derived_input_seed"
        ] = inputs[
            "derived_seed"
        ]

        mask = torch.from_numpy(
            inputs[
                "padding_mask"
            ]
        )


        if target == "STAGE20_MASKED_CNN_V1":

            module = import_path(
                "stage26_cnn_module",
                (
                    repo
                    / "scripts/stage20_masked_cnn.py"
                ),
            )

            model = (
                module.Stage20MaskedCNNv1()
            )

            state, load_mode, layout = (
                torch_load_state(
                    repo
                    / "results/stage20_1e_training/"
                      "stage20_1e2_epoch10_model_state_dict.pt"
                )
            )

            model.load_state_dict(
                state,
                strict=True,
            )

            model.eval()

            parameter_count = int(
                module.count_trainable_parameters(
                    model
                )
            )

            if parameter_count != 93025:
                raise RuntimeError(
                    f"CNN parameter count mismatch: {parameter_count}"
                )

            # Authentic frozen CNN boundary:
            # uint8 packet image + bool padding mask.
            # CNN itself performs float32 / 255 inside forward().
            image = torch.from_numpy(
                inputs[
                    "image_uint8"
                ]
            )

            result[
                "input_boundary"
            ] = (
                "UINT8_IMAGE__CNN_INTERNAL_FLOAT32_DIV255"
            )


        else:

            module = import_path(
                "stage26_vit_module",
                (
                    repo
                    / "scripts/stage21_masked_vit.py"
                ),
            )

            model = (
                module.Stage21MaskedViTv1()
            )

            state, load_mode, layout = (
                torch_load_state(
                    repo
                    / "results/stage21_architecture/"
                      "stage21_2_epoch10_model_state_dict.pt"
                )
            )

            model.load_state_dict(
                state,
                strict=True,
            )

            model.eval()

            parameter_count = int(
                module.count_trainable_parameters(
                    model
                )
            )

            if parameter_count != 91969:
                raise RuntimeError(
                    f"ViT parameter count mismatch: {parameter_count}"
                )

            # Authentic frozen ViT boundary:
            # already-scaled float32 image + bool mask.
            image = torch.from_numpy(
                inputs[
                    "image_scaled"
                ]
            )

            result[
                "input_boundary"
            ] = (
                "FLOAT32_DIV255_BEFORE_VIT_FORWARD"
            )


        result[
            "input_shape"
        ] = list(
            image.shape
        )

        result[
            "input_dtype"
        ] = str(
            image.dtype
        )

        result[
            "padding_mask_shape"
        ] = list(
            mask.shape
        )

        result[
            "padding_mask_dtype"
        ] = str(
            mask.dtype
        )

        with torch.inference_mode():

            logits = model(
                image,
                mask,
            )

            probabilities = torch.sigmoid(
                logits
            ).detach().cpu().numpy()

        p = validate_probability_vector(
            probabilities,
            expected_rows=batch_size,
        )

        result[
            "framework_version"
        ] = torch.__version__

        result[
            "model_class"
        ] = (
            model.__class__.__module__
            + "."
            + model.__class__.__name__
        )

        result[
            "parameter_count"
        ] = parameter_count

        result[
            "load_mode"
        ] = load_mode

        result[
            "state_layout"
        ] = layout

        result[
            "output_shape"
        ] = list(
            p.shape
        )

        result[
            "output_dtype"
        ] = str(
            p.dtype
        )

        result[
            "output_sha256"
        ] = sha256_array(
            p
        )


    else:
        raise RuntimeError(
            f"Unknown target: {target}"
        )


    # -------------------------------------------------------------------------
    # Universal final gates
    # -------------------------------------------------------------------------

    if result.get(
        "torch_num_threads"
    ) is not None:

        if (
            result[
                "torch_num_threads"
            ]
            !=
            thread_count
        ):
            raise RuntimeError(
                "PyTorch intra-op thread mismatch."
            )

        if (
            result[
                "torch_num_interop_threads"
            ]
            !=
            1
        ):
            raise RuntimeError(
                "PyTorch inter-op thread mismatch."
            )


    result[
        "status"
    ] = "PASS"

    result[
        "prediction_count"
    ] = int(
        batch_size
    )

    result[
        "timing_performed"
    ] = False

    result[
        "memory_profiling_performed"
    ] = False

    result[
        "holdout_accessed"
    ] = False

    result[
        "gpu_used"
    ] = False


    print(
        json.dumps(
            result,
            sort_keys=True,
        )
    )


if __name__ == "__main__":
    main()
'''


WORKER_PATH.write_text(
    textwrap.dedent(
        worker_source
    ).lstrip(),
    encoding="utf-8",
)


worker_sha = sha256_file(
    WORKER_PATH
)


print("Worker:")
print(" ", WORKER_PATH)

print("SHA256:")
print(" ", worker_sha)


# Explicit anti-timing source audit.
worker_text = WORKER_PATH.read_text(
    encoding="utf-8"
)


FORBIDDEN_SOURCE_TOKENS = [
    "perf_counter",
    "time.monotonic",
    "time.time(",
    "process_time(",
    "thread_time(",
    "cuda.Event",
]


bad_tokens = [
    token
    for token in FORBIDDEN_SOURCE_TOKENS
    if token in worker_text
]


if bad_tokens:
    raise RuntimeError(
        "Timing API accidentally present in preflight worker: "
        + repr(
            bad_tokens
        )
    )


print(
    "\n[PASS] No benchmark timing API found in preflight worker."
)


# =============================================================================
# 7. CREATE IMPLEMENTATION FREEZE RECORD
#
# This closes one under-specified implementation detail from the master lock:
# exact seed derivation. It is frozen here BEFORE any preflight prediction.
# =============================================================================

banner(
    "STAGE26-0D :: INPUT IMPLEMENTATION FREEZE"
)


implementation_record = {
    "schema":
        "stage26_0d_preflight_implementation_v1",

    "stage":
        26,

    "parent_protocol_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "measurement_seed":
        MEASUREMENT_SEED,

    "group_a_seed_derivation":
        "derived_seed = 26042 + batch_size",

    "group_b_seed_derivation":
        "derived_seed = 26042 + 1000000 + batch_size",

    "group_a_tree_input":
        (
            "float32(scaler.mean_ + Z * scaler.scale_)"
        ),

    "group_a_ft_input":
        "Z float32",

    "group_b_cnn_boundary":
        (
            "uint8 image + bool mask; frozen CNN forward performs "
            "float32 conversion and divide-by-255 internally"
        ),

    "group_b_vit_boundary":
        (
            "float32 image/255 + bool mask before frozen ViT forward"
        ),

    "preflight_batch_size":
        1,

    "preflight_prediction_per_condition":
        1,

    "timing_allowed":
        False,

    "memory_profiling_allowed":
        False,

    "holdout_access_allowed":
        False,

    "gpu_allowed":
        False,

    "worker_sha256":
        worker_sha,
}


IMPLEMENTATION_RECORD_PATH = (
    PREFLIGHT_LOCAL_DIR
    / "stage26_0d_preflight_implementation.json"
)


write_json(
    IMPLEMENTATION_RECORD_PATH,
    implementation_record,
)


implementation_sha = sha256_file(
    IMPLEMENTATION_RECORD_PATH
)


print("Implementation record:")
print(" ", IMPLEMENTATION_RECORD_PATH)

print("SHA256:")
print(" ", implementation_sha)


# =============================================================================
# 8. RUN 16 UNTIMERED PREFLIGHT SUBPROCESSES
# =============================================================================

banner(
    "STAGE26-0D :: RUN UNTIMERED CPU PREFLIGHT"
)


results = []

failures = []


condition_number = 0


for target in TARGETS:

    for (
        hardware_mode,
        mode,
    ) in CPU_MODES.items():

        condition_number += 1

        config = {
            "schema":
                "stage26_cpu_preflight_condition_v1",

            "repo":
                str(
                    REPO_DIR
                ),

            "target_id":
                target,

            "hardware_mode":
                hardware_mode,

            "thread_count":
                mode[
                    "threads"
                ],

            "affinity":
                mode[
                    "affinity"
                ],

            "batch_size":
                1,

            "measurement_seed":
                MEASUREMENT_SEED,

            "timing_allowed":
                False,

            "memory_profiling_allowed":
                False,
        }


        config_path = (
            PREFLIGHT_LOCAL_DIR
            / (
                f"preflight_{condition_number:02d}_"
                f"{target}_"
                f"{hardware_mode}.json"
            )
        )


        write_json(
            config_path,
            config,
        )


        env = os.environ.copy()


        thread_count = str(
            mode[
                "threads"
            ]
        )


        # Frozen thread variables passed BEFORE subprocess/framework imports.
        for key in [
            "OMP_NUM_THREADS",
            "MKL_NUM_THREADS",
            "OPENBLAS_NUM_THREADS",
            "NUMEXPR_NUM_THREADS",
            "BLIS_NUM_THREADS",
            "VECLIB_MAXIMUM_THREADS",
        ]:
            env[
                key
            ] = thread_count


        # Prevent accidental GPU path even if environment changes.
        env[
            "CUDA_VISIBLE_DEVICES"
        ] = ""


        p = subprocess.run(
            [
                sys.executable,
                str(
                    WORKER_PATH
                ),
                str(
                    config_path
                ),
            ],
            cwd=REPO_DIR,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=False,
        )


        if p.returncode != 0:

            failure = {
                "condition_number":
                    condition_number,

                "target_id":
                    target,

                "hardware_mode":
                    hardware_mode,

                "returncode":
                    p.returncode,

                "stdout":
                    p.stdout[-4000:],

                "stderr":
                    p.stderr[-8000:],
            }

            failures.append(
                failure
            )

            print(
                f"[FAIL] {condition_number:02d}/16 "
                f"{target:43s} "
                f"{hardware_mode}"
            )

            # Fail immediately. Do not continue discovering later
            # compatibility issues after an unresolved preflight failure.
            break


        lines = [
            line.strip()
            for line in p.stdout.splitlines()
            if line.strip()
        ]


        if not lines:
            raise RuntimeError(
                f"No worker JSON returned for {target}"
            )


        try:
            result = json.loads(
                lines[-1]
            )

        except Exception as exc:
            raise RuntimeError(
                "Unable to parse final worker output.\n\n"
                + p.stdout
                + "\n\nSTDERR:\n"
                + p.stderr
            ) from exc


        if result.get(
            "status"
        ) != "PASS":
            raise RuntimeError(
                f"Worker did not return PASS: {result}"
            )


        results.append(
            result
        )


        print(
            f"[PASS] {condition_number:02d}/16 "
            f"{target:43s} "
            f"{hardware_mode:21s} "
            f"output={result['output_sha256'][:16]}..."
        )


    if failures:
        break


# =============================================================================
# 9. FAILURE HANDLING
# =============================================================================

if failures:

    FAILURE_PATH = (
        PREFLIGHT_LOCAL_DIR
        / "stage26_0d_preflight_failure.json"
    )


    write_json(
        FAILURE_PATH,
        {
            "schema":
                "stage26_0d_preflight_failure_v1",

            "stage":
                26,

            "parent_protocol_commit":
                EXPECTED_HEAD,

            "timing_performed":
                False,

            "memory_profiling_performed":
                False,

            "holdout_accessed":
                False,

            "gpu_used":
                False,

            "failures":
                failures,
        },
    )


    print(
        "\nFailure receipt:"
    )

    print(
        " ",
        FAILURE_PATH,
    )


    first = failures[0]


    print(
        "\nSTDOUT:"
    )

    print(
        first[
            "stdout"
        ]
    )


    print(
        "\nSTDERR:"
    )

    print(
        first[
            "stderr"
        ]
    )


    raise RuntimeError(
        "STAGE26-0D PREFLIGHT FAILED. "
        "Do not begin benchmarking. "
        "Send the complete output for one scoped compatibility fix."
    )


# =============================================================================
# 10. CROSS-MODE DETERMINISM GATE
#
# Same model + same B=1 input should yield the same result fingerprint between
# the 1-core and 2-core CPU modes. This is NOT a performance comparison.
# =============================================================================

banner(
    "STAGE26-0D :: CROSS-MODE PREDICTION FINGERPRINT GATE"
)


by_target = {}


for result in results:

    by_target.setdefault(
        result[
            "target_id"
        ],
        {}
    )[
        result[
            "hardware_mode"
        ]
    ] = result


fingerprint_gate = []


for target in TARGETS:

    one = by_target[
        target
    ][
        "CPU_1_PHYSICAL_CORE"
    ]

    two = by_target[
        target
    ][
        "CPU_2_PHYSICAL_CORE"
    ]


    same = (
        one[
            "output_sha256"
        ]
        ==
        two[
            "output_sha256"
        ]
    )


    fingerprint_gate.append(
        {
            "target_id":
                target,

            "cpu1_sha256":
                one[
                    "output_sha256"
                ],

            "cpu2_sha256":
                two[
                    "output_sha256"
                ],

            "identical":
                same,
        }
    )


    print(
        f"{target:43s} "
        f"{'PASS' if same else 'FAIL'}"
    )


    if not same:
        raise RuntimeError(
            f"Prediction changed between CPU modes for {target}. "
            "Do not benchmark until resolved."
        )


# =============================================================================
# 11. PREFLIGHT RECEIPT
# =============================================================================

banner(
    "STAGE26-0D :: BUILD PREFLIGHT RECEIPT"
)


receipt = {
    "schema":
        "stage26_0d_cpu_preflight_receipt_v1",

    "stage":
        26,

    "status":
        "PASS",

    "completed_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_protocol_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "cpu_execution_plan_sha256":
        EXPECTED_PLAN_SHA256,

    "worker_sha256":
        worker_sha,

    "implementation_record_sha256":
        implementation_sha,

    "target_count":
        len(
            TARGETS
        ),

    "cpu_mode_count":
        len(
            CPU_MODES
        ),

    "preflight_condition_count":
        len(
            results
        ),

    "preflight_batch_size":
        1,

    "results":
        results,

    "cross_mode_fingerprint_gate":
        fingerprint_gate,

    "scientific_boundary_after_preflight": {
        "model_deserialization_permitted":
            True,

        "untimed_preflight_inference_performed":
            True,

        "performance_timing_performed":
            False,

        "latency_observations":
            0,

        "throughput_observations":
            0,

        "memory_measurements":
            0,

        "holdout_reopened":
            False,

        "gpu_used":
            False,

        "thresholds_changed":
            False,

        "models_changed":
            False,
    },

    "next_action":
        (
            "GIT_ANCHOR_STAGE26_0D_PREFLIGHT_BEFORE_FIRST_CPU_TIMING"
        ),
}


RECEIPT_PATH = (
    PREFLIGHT_LOCAL_DIR
    / "stage26_0d_cpu_preflight_receipt.json"
)


write_json(
    RECEIPT_PATH,
    receipt,
)


receipt_sha = sha256_file(
    RECEIPT_PATH
)


print("Preflight receipt:")
print(" ", RECEIPT_PATH)

print("SHA256:")
print(" ", receipt_sha)


# =============================================================================
# 12. COPY SUCCESSFUL PREFLIGHT PACKAGE INTO REPOSITORY
# =============================================================================

banner(
    "STAGE26-0D :: BUILD DURABLE PREFLIGHT PACKAGE"
)


if PREFLIGHT_REPO_DIR.exists():

    # This should not normally exist at this parent commit.
    if any(
        PREFLIGHT_REPO_DIR.iterdir()
    ):
        raise RuntimeError(
            "Stage26-0D repo package already exists unexpectedly."
        )

else:
    PREFLIGHT_REPO_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )


repo_worker = (
    PREFLIGHT_REPO_DIR
    / "stage26_cpu_preflight_worker.py"
)

repo_implementation = (
    PREFLIGHT_REPO_DIR
    / IMPLEMENTATION_RECORD_PATH.name
)

repo_receipt = (
    PREFLIGHT_REPO_DIR
    / RECEIPT_PATH.name
)


shutil.copy2(
    WORKER_PATH,
    repo_worker,
)

shutil.copy2(
    IMPLEMENTATION_RECORD_PATH,
    repo_implementation,
)

shutil.copy2(
    RECEIPT_PATH,
    repo_receipt,
)


# Package manifest.
PACKAGE_MANIFEST = (
    PREFLIGHT_REPO_DIR
    / "stage26_0d_preflight_package_manifest.json"
)


manifest_files = []


for path in sorted(
    PREFLIGHT_REPO_DIR.iterdir()
):

    if (
        path.is_file()
        and
        path.name
        !=
        PACKAGE_MANIFEST.name
    ):

        manifest_files.append(
            {
                "path":
                    str(
                        path.relative_to(
                            REPO_DIR
                        )
                    ),

                "size_bytes":
                    path.stat().st_size,

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )


package_manifest = {
    "schema":
        "stage26_0d_preflight_package_manifest_v1",

    "stage":
        26,

    "status":
        "READY_FOR_GIT_ANCHOR_BEFORE_FIRST_CPU_TIMING",

    "parent_protocol_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "preflight_receipt_sha256":
        receipt_sha,

    "files":
        manifest_files,

    "scientific_boundary": {
        "performance_timing_performed":
            False,

        "memory_profiled":
            False,

        "holdout_accessed":
            False,

        "gpu_used":
            False,
    },
}


write_json(
    PACKAGE_MANIFEST,
    package_manifest,
)


package_sha = sha256_file(
    PACKAGE_MANIFEST
)


print(
    "Repository package:"
)

print(
    " ",
    PREFLIGHT_REPO_DIR,
)

print(
    "\nPackage manifest SHA256:"
)

print(
    " ",
    package_sha,
)


# =============================================================================
# 13. FINAL GIT-STATE GATE
# =============================================================================

banner(
    "STAGE26-0D FINAL AUDIT"
)


STATUS_AFTER = git(
    "status",
    "--porcelain",
)


print(
    "Repository status:"
)

print(
    STATUS_AFTER
    if STATUS_AFTER
    else "<unexpected clean>"
)


if not STATUS_AFTER:
    raise RuntimeError(
        "Expected new Stage26-0D files."
    )


unexpected = []


for line in STATUS_AFTER.splitlines():

    path = line[3:]

    if not path.startswith(
        "results/stage26_deployment_profiling/stage26_0d_cpu_preflight/"
    ):
        unexpected.append(
            line
        )


if unexpected:
    raise RuntimeError(
        "Unexpected repository changes outside Stage26-0D:\n"
        + "\n".join(
            unexpected
        )
    )


# =============================================================================
# 14. CLOSURE
# =============================================================================

banner(
    "STAGE26-0D CPU PREFLIGHT COMPLETE"
)


print(
    "Protocol parent commit:"
)

print(
    " ",
    EXPECTED_HEAD,
)


print(
    "\nPreflight conditions:"
)

print(
    " ",
    len(
        results
    ),
    "/ 16 PASS",
)


print(
    "\nValidated targets:"
)

for target in TARGETS:
    print(
        "  PASS",
        target,
    )


print(
    "\nValidated CPU modes:"
)

print(
    "  PASS CPU_1_PHYSICAL_CORE affinity=[0] threads=1"
)

print(
    "  PASS CPU_2_PHYSICAL_CORE affinity=[0,1] threads=2"
)


print(
    "\nScientific state:"
)

print(
    "  MODEL DESERIALIZATION NOW VERIFIED"
)

print(
    "  EXACT FORWARD/PREDICT PATHS VERIFIED"
)

print(
    "  ONLY UNTIMERED BATCH-1 PREFLIGHT PREDICTIONS EXECUTED"
)

print(
    "  NO PERFORMANCE CLOCK USED"
)

print(
    "  LATENCY OBSERVATIONS = 0"
)

print(
    "  THROUGHPUT OBSERVATIONS = 0"
)

print(
    "  MEMORY MEASUREMENTS = 0"
)

print(
    "  HOLDOUT NOT REOPENED"
)

print(
    "  GPU NOT USED"
)


print(
    "\nPREFLIGHT RECEIPT SHA256:"
)

print(
    " ",
    receipt_sha,
)


print(
    "\nPACKAGE MANIFEST SHA256:"
)

print(
    " ",
    package_sha,
)


print(
    "\nCRITICAL NEXT ACTION:"
)

print(
    "  STAGE26-0D-GIT — COMMIT + PUSH + REMOTE VERIFY PREFLIGHT PACKAGE."
)

print(
    "  DO NOT START CPU TIMING UNTIL THAT REMOTE CHECK PASSES."
)


STAGE26-0D :: SCIENTIFIC / GIT GATE
Expected HEAD : f3957b964e659392c57d1b12881163288f2efedc
Local HEAD    : f3957b964e659392c57d1b12881163288f2efedc
origin/main   : f3957b964e659392c57d1b12881163288f2efedc
Repo clean    : True

measurement_protocol SHA: d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
execution_plan SHA      : b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363

STAGE26-0D :: SOURCE / ARTIFACT GATE
[OK] xgboost            results/stage16_classical_benchmark_checkpoint/stage16_3_tuned_models/XGBOOST_tuned.joblib
[OK] lightgbm           results/stage16_classical_benchmark_checkpoint/stage16_3_tuned_models/LIGHTGBM_tuned.joblib
[OK] catboost           results/stage16_classical_benchmark_checkpoint/stage16_3_tuned_models/CATBOOST_tuned.joblib
[OK] ft_seed7           results/stage15_transformer_checkpoint/stage15_4b_models/FT_BALANCED_seed_7_best_extended.pt
[OK] ft_seed29          results/stage15_transformer_checkpoint/stage15_4a_models/FT_BA

RuntimeError: Timing API accidentally present in preflight worker: ['perf_counter']

In [12]:
# =============================================================================
# STAGE26-0D — CPU EXECUTION PREFLIGHT
#
# PURPOSE
# -------
# Validate that every frozen Stage26 target can be deserialized and can execute
# its exact CPU inference path under BOTH frozen CPU execution modes.
#
# SCIENTIFIC BOUNDARY
# -------------------
# ALLOWED:
#   - model deserialization
#   - deterministic synthetic input construction
#   - exactly one untimed prediction per target × CPU mode
#   - output shape / finiteness validation
#   - architecture parameter-count validation
#   - prediction fingerprinting (SHA256 only; values not inspected)
#
# FORBIDDEN:
#   - perf_counter / monotonic / time.time / CUDA events
#   - latency measurement
#   - throughput measurement
#   - memory profiling
#   - warmup loops
#   - timed loops
#   - holdout access
#   - model selection / threshold tuning
#   - GPU
#
# TARGETS:
#   8 deployment targets × 2 CPU modes = 16 preflight executions.
#
# NOTE:
#   The preflight uses fresh subprocesses so thread environment variables and
#   CPU affinity are applied before ML/numerical framework import.
# =============================================================================

from __future__ import annotations

import os
import sys
import json
import stat
import shutil
import hashlib
import subprocess
import textwrap
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. PATHS / FROZEN IDENTITIES
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

WORKERS_DIR = (
    STAGE26_ROOT
    / "workers"
)

RUNTIME_DIR = (
    STAGE26_ROOT
    / "runtime"
)

PROTOCOL_DIR = (
    STAGE26_ROOT
    / "protocol"
)

PREFLIGHT_LOCAL_DIR = (
    STAGE26_ROOT
    / "preflight"
)

PREFLIGHT_REPO_DIR = (
    REPO_DIR
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0d_cpu_preflight"
)

for d in [
    WORKERS_DIR,
    RUNTIME_DIR,
    PROTOCOL_DIR,
    PREFLIGHT_LOCAL_DIR,
]:
    d.mkdir(
        parents=True,
        exist_ok=True,
    )


EXPECTED_HEAD = (
    "f3957b964e659392c57d1b12881163288f2efedc"
)

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

EXPECTED_PLAN_SHA256 = (
    "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363"
)

MEASUREMENT_SEED = 26042


# =============================================================================
# 1. HELPERS — NO TIMING FUNCTIONS
# =============================================================================

def banner(text: str):
    print("\n" + "=" * 104)
    print(text)
    print("=" * 104)


def git(*args):
    p = subprocess.run(
        ["git", *args],
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(path: Path):
    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def write_json(path: Path, payload):
    path.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )


# =============================================================================
# 2. PRE-PREFLIGHT SCIENTIFIC GATE
# =============================================================================

banner(
    "STAGE26-0D :: SCIENTIFIC / GIT GATE"
)

HEAD = git(
    "rev-parse",
    "HEAD",
)

REMOTE = git(
    "rev-parse",
    "origin/main",
)

STATUS = git(
    "status",
    "--porcelain",
)

print("Expected HEAD :", EXPECTED_HEAD)
print("Local HEAD    :", HEAD)
print("origin/main   :", REMOTE)
print("Repo clean    :", STATUS == "")


if HEAD != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected local HEAD."
    )

if REMOTE != EXPECTED_HEAD:
    raise RuntimeError(
        "origin/main no longer matches the frozen Stage26 protocol commit."
    )

if STATUS:
    raise RuntimeError(
        "Repository must be clean before CPU preflight:\n"
        + STATUS
    )


MEASUREMENT_PROTOCOL = (
    REPO_DIR
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXECUTION_PLAN = (
    REPO_DIR
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "cpu_execution_plan.json"
)


protocol_sha = sha256_file(
    MEASUREMENT_PROTOCOL
)

plan_sha = sha256_file(
    EXECUTION_PLAN
)


print(
    "\nmeasurement_protocol SHA:",
    protocol_sha,
)

print(
    "execution_plan SHA      :",
    plan_sha,
)


if protocol_sha != EXPECTED_PROTOCOL_SHA256:
    raise RuntimeError(
        "Frozen measurement protocol SHA mismatch."
    )

if plan_sha != EXPECTED_PLAN_SHA256:
    raise RuntimeError(
        "Frozen execution-plan SHA mismatch."
    )


# =============================================================================
# 3. FROZEN SOURCE / ARTIFACT PATHS
# =============================================================================

banner(
    "STAGE26-0D :: SOURCE / ARTIFACT GATE"
)


PATHS = {
    # -------------------------------------------------------------------------
    # Classical models
    # -------------------------------------------------------------------------
    "xgboost": (
        REPO_DIR
        / "results/stage16_classical_benchmark_checkpoint/"
          "stage16_3_tuned_models/XGBOOST_tuned.joblib"
    ),

    "lightgbm": (
        REPO_DIR
        / "results/stage16_classical_benchmark_checkpoint/"
          "stage16_3_tuned_models/LIGHTGBM_tuned.joblib"
    ),

    "catboost": (
        REPO_DIR
        / "results/stage16_classical_benchmark_checkpoint/"
          "stage16_3_tuned_models/CATBOOST_tuned.joblib"
    ),

    # -------------------------------------------------------------------------
    # Stage15 FT checkpoints
    # -------------------------------------------------------------------------
    "ft_seed7": (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_4b_models/FT_BALANCED_seed_7_best_extended.pt"
    ),

    "ft_seed29": (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_4a_models/FT_BALANCED_seed_29_best.pt"
    ),

    "ft_seed101": (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_4a_models/FT_BALANCED_seed_101_best.pt"
    ),

    "ft_seed313": (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_4c_models/FT_BALANCED_seed_313_best.pt"
    ),

    "ft_seed997": (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_4c_models/FT_BALANCED_seed_997_best.pt"
    ),

    "ft_module": (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "ft_transformer_numeric.py"
    ),

    "ft_architecture": (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_4c_frozen_architecture.json"
    ),

    "scaler": (
        REPO_DIR
        / "results/stage15_transformer_checkpoint/"
          "stage15_2_standard_scaler.joblib"
    ),

    # -------------------------------------------------------------------------
    # Packet models
    # -------------------------------------------------------------------------
    "cnn_state": (
        REPO_DIR
        / "results/stage20_1e_training/"
          "stage20_1e2_epoch10_model_state_dict.pt"
    ),

    "cnn_module": (
        REPO_DIR
        / "scripts/stage20_masked_cnn.py"
    ),

    "vit_state": (
        REPO_DIR
        / "results/stage21_architecture/"
          "stage21_2_epoch10_model_state_dict.pt"
    ),

    "vit_module": (
        REPO_DIR
        / "scripts/stage21_masked_vit.py"
    ),

    "packet_encoder": (
        REPO_DIR
        / "scripts/stage20_packet_image_encoder.py"
    ),
}


for name, path in PATHS.items():

    if not path.exists():
        raise FileNotFoundError(
            f"{name}: {path}"
        )

    print(
        f"[OK] {name:18s} "
        f"{path.relative_to(REPO_DIR)}"
    )


# =============================================================================
# 4. SOURCE-CODE SHA GATES
# =============================================================================

banner(
    "STAGE26-0D :: EXECUTABLE SOURCE SHA GATE"
)


SOURCE_HASHES = {
    "ft_module": (
        "8233a7b62e7045d7c920c15fd8ec974ed8b3db6f2f131dc4c64ba0ce0133dcfc"
    ),

    "cnn_module": (
        "3638ae622017a36e6eeb33f227135829695ff2f3581c9b43787a02c1a440b9d4"
    ),

    "vit_module": (
        "3af99e4ea7061c68a676dc8fa7e485a7d13278f8947e4f8a8fbf2069dc31e3cb"
    ),

    "packet_encoder": (
        "9883fe2b27020aaff707a753123b35eb3223d21abf295d056ec233e532f94222"
    ),

    "scaler": (
        "ac1e3a9b0a2409edcd98c293f31c22b9c140cc882518881565d16eb1e2a573b5"
    ),
}


for name, expected in SOURCE_HASHES.items():

    actual = sha256_file(
        PATHS[name]
    )

    ok = (
        actual
        ==
        expected
    )

    print(
        f"{name:18s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual}"
    )

    if not ok:
        raise RuntimeError(
            f"Source/dependency SHA mismatch: {name}"
        )


# =============================================================================
# 5. FROZEN CPU MODES
# =============================================================================

CPU_MODES = {
    "CPU_1_PHYSICAL_CORE": {
        "threads": 1,
        "affinity": [0],
    },

    "CPU_2_PHYSICAL_CORE": {
        "threads": 2,
        "affinity": [0, 1],
    },
}


TARGETS = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
    "ENS_LGBM_XGB_EQUAL",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
]


print(
    "\nPreflight targets:",
    len(TARGETS),
)

print(
    "CPU modes:",
    len(CPU_MODES),
)

print(
    "Total subprocess checks:",
    len(TARGETS) * len(CPU_MODES),
)


# =============================================================================
# 6. CREATE PREFLIGHT WORKER
#
# IMPORTANT:
#   This worker intentionally contains NO performance clock.
# =============================================================================

banner(
    "STAGE26-0D :: CREATE UNTIMERED PREFLIGHT WORKER"
)


WORKER_PATH = (
    WORKERS_DIR
    / "stage26_cpu_preflight_worker.py"
)


worker_source = r'''
from __future__ import annotations

import os
import sys
import json
import hashlib
import importlib.util
from pathlib import Path


# =============================================================================
# No performance timing is permitted in this worker.
# =============================================================================

FORBIDDEN_CLOCK_NAMES = [
    "perf_counter",
    "monotonic",
    "process_time",
    "thread_time",
    "time_ns",
]


def sha256_array(array):
    import numpy as np

    a = np.ascontiguousarray(
        array
    )

    h = hashlib.sha256()

    h.update(
        str(
            a.dtype
        ).encode("ascii")
    )

    h.update(
        json.dumps(
            list(
                a.shape
            )
        ).encode("ascii")
    )

    h.update(
        a.tobytes(
            order="C"
        )
    )

    return h.hexdigest()


def import_path(name, path):

    spec = importlib.util.spec_from_file_location(
        name,
        str(path),
    )

    if spec is None or spec.loader is None:
        raise RuntimeError(
            f"Unable to import {path}"
        )

    module = importlib.util.module_from_spec(
        spec
    )

    spec.loader.exec_module(
        module
    )

    return module


def extract_torch_state(obj):

    import torch

    if not isinstance(
        obj,
        dict,
    ):
        raise TypeError(
            f"Checkpoint root type unsupported: {type(obj)}"
        )

    # Direct state_dict.
    if obj and all(
        torch.is_tensor(v)
        for v in obj.values()
    ):
        return (
            obj,
            "DIRECT_STATE_DICT",
        )

    for key in [
        "model_state_dict",
        "state_dict",
        "model_state",
        "network_state_dict",
        "model",
    ]:

        candidate = obj.get(
            key
        )

        if (
            isinstance(
                candidate,
                dict,
            )
            and
            candidate
            and
            all(
                torch.is_tensor(v)
                for v in candidate.values()
            )
        ):
            return (
                candidate,
                key,
            )

    raise RuntimeError(
        "Unable to identify torch state_dict layout. "
        f"Top-level keys={list(obj.keys())[:30]}"
    )


def torch_load_state(path):

    import torch

    try:
        obj = torch.load(
            str(path),
            map_location="cpu",
            weights_only=True,
        )

        load_mode = (
            "torch.load(weights_only=True)"
        )

    except TypeError:

        obj = torch.load(
            str(path),
            map_location="cpu",
        )

        load_mode = (
            "torch.load(default)"
        )

    state, state_layout = (
        extract_torch_state(
            obj
        )
    )

    return (
        state,
        load_mode,
        state_layout,
    )


def configure_torch_threads(thread_count):

    import torch

    torch.set_num_threads(
        int(
            thread_count
        )
    )

    try:
        torch.set_num_interop_threads(
            1
        )
    except RuntimeError:
        # This is still checked below. A mismatch fails the preflight.
        pass

    return {
        "torch_num_threads":
            int(
                torch.get_num_threads()
            ),

        "torch_num_interop_threads":
            int(
                torch.get_num_interop_threads()
            ),
    }


def build_group_a_input(
    *,
    repo,
    batch_size,
    seed,
):

    import joblib
    import numpy as np

    scaler_path = (
        repo
        / "results/stage15_transformer_checkpoint/"
          "stage15_2_standard_scaler.joblib"
    )

    scaler = joblib.load(
        scaler_path
    )

    if int(
        scaler.n_features_in_
    ) != 70:
        raise RuntimeError(
            "Frozen scaler does not have 70 features."
        )

    # Frozen implementation of:
    # "deterministic from frozen seed derived from batch size"
    #
    # No performance result is consulted.
    derived_seed = (
        int(seed)
        +
        int(batch_size)
    )

    rng = np.random.default_rng(
        derived_seed
    )

    Z = rng.normal(
        loc=0.0,
        scale=1.0,
        size=(
            int(batch_size),
            70,
        ),
    ).astype(
        np.float32
    )

    mean = np.asarray(
        scaler.mean_,
        dtype=np.float32,
    )

    scale = np.asarray(
        scaler.scale_,
        dtype=np.float32,
    )

    X_raw = (
        mean[None, :]
        +
        Z
        *
        scale[None, :]
    ).astype(
        np.float32,
        copy=False,
    )

    if not np.isfinite(
        Z
    ).all():
        raise RuntimeError(
            "Non-finite FT synthetic input."
        )

    if not np.isfinite(
        X_raw
    ).all():
        raise RuntimeError(
            "Non-finite tree synthetic input."
        )

    return {
        "Z": Z,
        "X_raw": X_raw,
        "derived_seed": derived_seed,
    }


def make_ipv4_packet(
    rng,
    protocol,
    length,
):

    import numpy as np

    if length < 40:
        raise ValueError(
            "Synthetic packet length must be >=40."
        )

    packet = bytearray(
        rng.integers(
            0,
            256,
            size=int(length),
            dtype=np.uint8,
        ).tobytes()
    )

    # IPv4 + IHL 5.
    packet[0] = 0x45

    # Total length.
    packet[2:4] = int(
        length
    ).to_bytes(
        2,
        "big",
    )

    # Fragment offset = zero.
    packet[6] = (
        packet[6]
        &
        0xE0
    )

    packet[7] = 0

    # Protocol 6 TCP / 17 UDP.
    packet[9] = int(
        protocol
    )

    return bytes(
        packet
    )


def build_packet_input(
    *,
    repo,
    batch_size,
    seed,
):

    import numpy as np

    encoder = import_path(
        "stage26_packet_encoder",
        (
            repo
            / "scripts/stage20_packet_image_encoder.py"
        ),
    )

    derived_seed = (
        int(seed)
        +
        1_000_000
        +
        int(batch_size)
    )

    rng = np.random.default_rng(
        derived_seed
    )

    images_uint8 = np.zeros(
        (
            int(batch_size),
            1,
            64,
            256,
        ),
        dtype=np.uint8,
    )

    masks = np.zeros(
        (
            int(batch_size),
            1,
            64,
            256,
        ),
        dtype=np.bool_,
    )

    for flow_idx in range(
        int(batch_size)
    ):

        packet_count = int(
            rng.integers(
                1,
                65,
            )
        )

        packets = []

        for _ in range(
            packet_count
        ):

            protocol = (
                6
                if int(
                    rng.integers(
                        0,
                        2,
                    )
                ) == 0
                else 17
            )

            packet_length = int(
                rng.integers(
                    40,
                    257,
                )
            )

            packets.append(
                make_ipv4_packet(
                    rng,
                    protocol,
                    packet_length,
                )
            )

        image, mask = (
            encoder.encode_flow(
                packets
            )
        )

        images_uint8[
            flow_idx,
            0,
        ] = image

        masks[
            flow_idx,
            0,
        ] = mask

    images_scaled = (
        images_uint8.astype(
            np.float32
        )
        /
        np.float32(
            255.0
        )
    )

    return {
        "image_uint8":
            images_uint8,

        "image_scaled":
            images_scaled,

        "padding_mask":
            masks,

        "derived_seed":
            derived_seed,
    }


def validate_probability_vector(
    probabilities,
    *,
    expected_rows,
):

    import numpy as np

    p = np.asarray(
        probabilities
    )

    p = p.reshape(
        -1
    )

    if p.shape != (
        int(expected_rows),
    ):
        raise RuntimeError(
            f"Unexpected probability shape: {p.shape}"
        )

    if not np.isfinite(
        p
    ).all():
        raise RuntimeError(
            "Non-finite prediction encountered."
        )

    if (
        (p < 0.0).any()
        or
        (p > 1.0).any()
    ):
        raise RuntimeError(
            "Probability outside [0,1]."
        )

    return p


def load_ft_model(
    *,
    repo,
    checkpoint_path,
):

    import torch

    ft_module = import_path(
        "stage26_ft_transformer",
        (
            repo
            / "results/stage15_transformer_checkpoint/"
              "ft_transformer_numeric.py"
        ),
    )

    arch_path = (
        repo
        / "results/stage15_transformer_checkpoint/"
          "stage15_4c_frozen_architecture.json"
    )

    arch_record = json.loads(
        arch_path.read_text(
            encoding="utf-8"
        )
    )

    arch = arch_record[
        "architecture"
    ]

    model = ft_module.NumericFTTransformer(
        n_features=int(
            arch_record[
                "input_predictor_count"
            ]
        ),
        d_token=int(
            arch["d_token"]
        ),
        n_heads=int(
            arch["n_heads"]
        ),
        n_layers=int(
            arch["n_layers"]
        ),
        d_ff=int(
            arch["d_ff"]
        ),
        dropout=float(
            arch["dropout"]
        ),
    )

    state, load_mode, layout = (
        torch_load_state(
            checkpoint_path
        )
    )

    model.load_state_dict(
        state,
        strict=True,
    )

    model.eval()

    parameter_count = int(
        sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        )
    )

    if parameter_count != 159169:
        raise RuntimeError(
            "FT parameter count mismatch: "
            f"{parameter_count}"
        )

    return (
        model,
        parameter_count,
        load_mode,
        layout,
    )


def main():

    config_path = Path(
        sys.argv[1]
    )

    config = json.loads(
        config_path.read_text(
            encoding="utf-8"
        )
    )

    repo = Path(
        config["repo"]
    )

    target = config[
        "target_id"
    ]

    hardware_mode = config[
        "hardware_mode"
    ]

    thread_count = int(
        config["thread_count"]
    )

    affinity = [
        int(x)
        for x in config[
            "affinity"
        ]
    ]

    batch_size = int(
        config.get(
            "batch_size",
            1,
        )
    )

    seed = int(
        config["measurement_seed"]
    )


    # -------------------------------------------------------------------------
    # Freeze CPU affinity BEFORE framework import.
    # -------------------------------------------------------------------------

    if hasattr(
        os,
        "sched_setaffinity",
    ):
        os.sched_setaffinity(
            0,
            set(
                affinity
            ),
        )

    observed_affinity = (
        sorted(
            os.sched_getaffinity(
                0
            )
        )
        if hasattr(
            os,
            "sched_getaffinity",
        )
        else None
    )


    result = {
        "status": "STARTED",
        "target_id": target,
        "hardware_mode": hardware_mode,
        "thread_count_requested": thread_count,
        "affinity_requested": affinity,
        "affinity_observed": observed_affinity,
        "batch_size": batch_size,
        "timing_performed": False,
        "memory_profiling_performed": False,
        "holdout_accessed": False,
        "gpu_used": False,
    }


    if (
        observed_affinity is not None
        and
        observed_affinity != affinity
    ):
        raise RuntimeError(
            f"Affinity mismatch: {observed_affinity} != {affinity}"
        )


    # -------------------------------------------------------------------------
    # Group A classical models
    # -------------------------------------------------------------------------

    if target in {
        "STAGE16_XGBOOST_TUNED",
        "STAGE16_LIGHTGBM_TUNED",
        "STAGE16_CATBOOST_TUNED",
        "ENS_LGBM_XGB_EQUAL",
    }:

        import joblib
        import numpy as np

        inputs = build_group_a_input(
            repo=repo,
            batch_size=batch_size,
            seed=seed,
        )

        X = inputs[
            "X_raw"
        ]

        result[
            "derived_input_seed"
        ] = inputs[
            "derived_seed"
        ]

        result[
            "input_shape"
        ] = list(
            X.shape
        )

        result[
            "input_dtype"
        ] = str(
            X.dtype
        )


        if target == "STAGE16_XGBOOST_TUNED":

            import xgboost

            path = (
                repo
                / "results/stage16_classical_benchmark_checkpoint/"
                  "stage16_3_tuned_models/XGBOOST_tuned.joblib"
            )

            model = joblib.load(
                path
            )

            if hasattr(
                model,
                "set_params",
            ):
                model.set_params(
                    n_jobs=thread_count
                )

            probabilities = (
                model.predict_proba(
                    X
                )[:, 1]
            )

            result[
                "framework_version"
            ] = xgboost.__version__

            result[
                "model_class"
            ] = (
                model.__class__.__module__
                + "."
                + model.__class__.__name__
            )


        elif target == "STAGE16_LIGHTGBM_TUNED":

            import lightgbm

            path = (
                repo
                / "results/stage16_classical_benchmark_checkpoint/"
                  "stage16_3_tuned_models/LIGHTGBM_tuned.joblib"
            )

            model = joblib.load(
                path
            )

            if hasattr(
                model,
                "set_params",
            ):
                model.set_params(
                    n_jobs=thread_count
                )

            probabilities = (
                model.predict_proba(
                    X
                )[:, 1]
            )

            result[
                "framework_version"
            ] = lightgbm.__version__

            result[
                "model_class"
            ] = (
                model.__class__.__module__
                + "."
                + model.__class__.__name__
            )


        elif target == "STAGE16_CATBOOST_TUNED":

            import catboost

            path = (
                repo
                / "results/stage16_classical_benchmark_checkpoint/"
                  "stage16_3_tuned_models/CATBOOST_tuned.joblib"
            )

            model = joblib.load(
                path
            )

            probabilities = (
                model.predict_proba(
                    X,
                    thread_count=thread_count,
                )[:, 1]
            )

            result[
                "framework_version"
            ] = catboost.__version__

            result[
                "model_class"
            ] = (
                model.__class__.__module__
                + "."
                + model.__class__.__name__
            )


        else:

            import xgboost
            import lightgbm

            xgb_path = (
                repo
                / "results/stage16_classical_benchmark_checkpoint/"
                  "stage16_3_tuned_models/XGBOOST_tuned.joblib"
            )

            lgb_path = (
                repo
                / "results/stage16_classical_benchmark_checkpoint/"
                  "stage16_3_tuned_models/LIGHTGBM_tuned.joblib"
            )

            xgb_model = joblib.load(
                xgb_path
            )

            lgb_model = joblib.load(
                lgb_path
            )

            if hasattr(
                xgb_model,
                "set_params",
            ):
                xgb_model.set_params(
                    n_jobs=thread_count
                )

            if hasattr(
                lgb_model,
                "set_params",
            ):
                lgb_model.set_params(
                    n_jobs=thread_count
                )

            xgb_p = (
                xgb_model.predict_proba(
                    X
                )[:, 1]
            )

            lgb_p = (
                lgb_model.predict_proba(
                    X
                )[:, 1]
            )

            probabilities = (
                np.asarray(
                    xgb_p,
                    dtype=np.float64,
                )
                +
                np.asarray(
                    lgb_p,
                    dtype=np.float64,
                )
            ) / 2.0

            result[
                "framework_version"
            ] = {
                "xgboost":
                    xgboost.__version__,
                "lightgbm":
                    lightgbm.__version__,
            }

            result[
                "model_class"
            ] = (
                "EqualWeight("
                + xgb_model.__class__.__name__
                + ","
                + lgb_model.__class__.__name__
                + ")"
            )


        p = validate_probability_vector(
            probabilities,
            expected_rows=batch_size,
        )

        result[
            "output_shape"
        ] = list(
            p.shape
        )

        result[
            "output_dtype"
        ] = str(
            p.dtype
        )

        result[
            "output_sha256"
        ] = sha256_array(
            p
        )


    # -------------------------------------------------------------------------
    # FT models
    # -------------------------------------------------------------------------

    elif target in {
        "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
        "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
    }:

        import numpy as np
        import torch

        thread_receipt = (
            configure_torch_threads(
                thread_count
            )
        )

        result.update(
            thread_receipt
        )

        inputs = build_group_a_input(
            repo=repo,
            batch_size=batch_size,
            seed=seed,
        )

        Z = inputs[
            "Z"
        ]

        result[
            "derived_input_seed"
        ] = inputs[
            "derived_seed"
        ]

        result[
            "input_shape"
        ] = list(
            Z.shape
        )

        result[
            "input_dtype"
        ] = str(
            Z.dtype
        )

        x = torch.from_numpy(
            Z
        )

        seed_paths = {
            7:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4b_models/FT_BALANCED_seed_7_best_extended.pt",

            29:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4a_models/FT_BALANCED_seed_29_best.pt",

            101:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4a_models/FT_BALANCED_seed_101_best.pt",

            313:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4c_models/FT_BALANCED_seed_313_best.pt",

            997:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4c_models/FT_BALANCED_seed_997_best.pt",
        }

        seeds = (
            [7, 29, 101, 313, 997]
            if target
            ==
            "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING"
            else
            [7]
        )

        member_probabilities = []

        load_receipts = []

        for checkpoint_seed in seeds:

            (
                model,
                parameter_count,
                load_mode,
                layout,
            ) = load_ft_model(
                repo=repo,
                checkpoint_path=seed_paths[
                    checkpoint_seed
                ],
            )

            with torch.inference_mode():

                logits = model(
                    x
                )

                member_p = torch.sigmoid(
                    logits
                ).detach().cpu().numpy()

            member_probabilities.append(
                member_p
            )

            load_receipts.append(
                {
                    "seed":
                        checkpoint_seed,

                    "parameter_count":
                        parameter_count,

                    "load_mode":
                        load_mode,

                    "state_layout":
                        layout,
                }
            )

        probabilities = np.mean(
            np.stack(
                member_probabilities,
                axis=0,
            ),
            axis=0,
        )

        p = validate_probability_vector(
            probabilities,
            expected_rows=batch_size,
        )

        result[
            "framework_version"
        ] = torch.__version__

        result[
            "model_class"
        ] = (
            "NumericFTTransformer"
        )

        result[
            "checkpoint_count"
        ] = len(
            seeds
        )

        result[
            "load_receipts"
        ] = load_receipts

        result[
            "output_shape"
        ] = list(
            p.shape
        )

        result[
            "output_dtype"
        ] = str(
            p.dtype
        )

        result[
            "output_sha256"
        ] = sha256_array(
            p
        )


    # -------------------------------------------------------------------------
    # CNN / ViT
    # -------------------------------------------------------------------------

    elif target in {
        "STAGE20_MASKED_CNN_V1",
        "STAGE21_MASKED_VIT_V1",
    }:

        import numpy as np
        import torch

        thread_receipt = (
            configure_torch_threads(
                thread_count
            )
        )

        result.update(
            thread_receipt
        )

        inputs = build_packet_input(
            repo=repo,
            batch_size=batch_size,
            seed=seed,
        )

        result[
            "derived_input_seed"
        ] = inputs[
            "derived_seed"
        ]

        mask = torch.from_numpy(
            inputs[
                "padding_mask"
            ]
        )


        if target == "STAGE20_MASKED_CNN_V1":

            module = import_path(
                "stage26_cnn_module",
                (
                    repo
                    / "scripts/stage20_masked_cnn.py"
                ),
            )

            model = (
                module.Stage20MaskedCNNv1()
            )

            state, load_mode, layout = (
                torch_load_state(
                    repo
                    / "results/stage20_1e_training/"
                      "stage20_1e2_epoch10_model_state_dict.pt"
                )
            )

            model.load_state_dict(
                state,
                strict=True,
            )

            model.eval()

            parameter_count = int(
                module.count_trainable_parameters(
                    model
                )
            )

            if parameter_count != 93025:
                raise RuntimeError(
                    f"CNN parameter count mismatch: {parameter_count}"
                )

            # Authentic frozen CNN boundary:
            # uint8 packet image + bool padding mask.
            # CNN itself performs float32 / 255 inside forward().
            image = torch.from_numpy(
                inputs[
                    "image_uint8"
                ]
            )

            result[
                "input_boundary"
            ] = (
                "UINT8_IMAGE__CNN_INTERNAL_FLOAT32_DIV255"
            )


        else:

            module = import_path(
                "stage26_vit_module",
                (
                    repo
                    / "scripts/stage21_masked_vit.py"
                ),
            )

            model = (
                module.Stage21MaskedViTv1()
            )

            state, load_mode, layout = (
                torch_load_state(
                    repo
                    / "results/stage21_architecture/"
                      "stage21_2_epoch10_model_state_dict.pt"
                )
            )

            model.load_state_dict(
                state,
                strict=True,
            )

            model.eval()

            parameter_count = int(
                module.count_trainable_parameters(
                    model
                )
            )

            if parameter_count != 91969:
                raise RuntimeError(
                    f"ViT parameter count mismatch: {parameter_count}"
                )

            # Authentic frozen ViT boundary:
            # already-scaled float32 image + bool mask.
            image = torch.from_numpy(
                inputs[
                    "image_scaled"
                ]
            )

            result[
                "input_boundary"
            ] = (
                "FLOAT32_DIV255_BEFORE_VIT_FORWARD"
            )


        result[
            "input_shape"
        ] = list(
            image.shape
        )

        result[
            "input_dtype"
        ] = str(
            image.dtype
        )

        result[
            "padding_mask_shape"
        ] = list(
            mask.shape
        )

        result[
            "padding_mask_dtype"
        ] = str(
            mask.dtype
        )

        with torch.inference_mode():

            logits = model(
                image,
                mask,
            )

            probabilities = torch.sigmoid(
                logits
            ).detach().cpu().numpy()

        p = validate_probability_vector(
            probabilities,
            expected_rows=batch_size,
        )

        result[
            "framework_version"
        ] = torch.__version__

        result[
            "model_class"
        ] = (
            model.__class__.__module__
            + "."
            + model.__class__.__name__
        )

        result[
            "parameter_count"
        ] = parameter_count

        result[
            "load_mode"
        ] = load_mode

        result[
            "state_layout"
        ] = layout

        result[
            "output_shape"
        ] = list(
            p.shape
        )

        result[
            "output_dtype"
        ] = str(
            p.dtype
        )

        result[
            "output_sha256"
        ] = sha256_array(
            p
        )


    else:
        raise RuntimeError(
            f"Unknown target: {target}"
        )


    # -------------------------------------------------------------------------
    # Universal final gates
    # -------------------------------------------------------------------------

    if result.get(
        "torch_num_threads"
    ) is not None:

        if (
            result[
                "torch_num_threads"
            ]
            !=
            thread_count
        ):
            raise RuntimeError(
                "PyTorch intra-op thread mismatch."
            )

        if (
            result[
                "torch_num_interop_threads"
            ]
            !=
            1
        ):
            raise RuntimeError(
                "PyTorch inter-op thread mismatch."
            )


    result[
        "status"
    ] = "PASS"

    result[
        "prediction_count"
    ] = int(
        batch_size
    )

    result[
        "timing_performed"
    ] = False

    result[
        "memory_profiling_performed"
    ] = False

    result[
        "holdout_accessed"
    ] = False

    result[
        "gpu_used"
    ] = False


    print(
        json.dumps(
            result,
            sort_keys=True,
        )
    )


if __name__ == "__main__":
    main()
'''


WORKER_PATH.write_text(
    textwrap.dedent(
        worker_source
    ).lstrip(),
    encoding="utf-8",
)


worker_sha = sha256_file(
    WORKER_PATH
)


print("Worker:")
print(" ", WORKER_PATH)

print("SHA256:")
print(" ", worker_sha)


# Explicit anti-timing source audit.
worker_text = WORKER_PATH.read_text(
    encoding="utf-8"
)


FORBIDDEN_SOURCE_TOKENS = [
    "perf_counter(",
    "time.monotonic",
    "time.time(",
    "process_time(",
    "thread_time(",
    "cuda.Event",
]


bad_tokens = [
    token
    for token in FORBIDDEN_SOURCE_TOKENS
    if token in worker_text
]


if bad_tokens:
    raise RuntimeError(
        "Timing API accidentally present in preflight worker: "
        + repr(
            bad_tokens
        )
    )


print(
    "\n[PASS] No benchmark timing API found in preflight worker."
)


# =============================================================================
# 7. CREATE IMPLEMENTATION FREEZE RECORD
#
# This closes one under-specified implementation detail from the master lock:
# exact seed derivation. It is frozen here BEFORE any preflight prediction.
# =============================================================================

banner(
    "STAGE26-0D :: INPUT IMPLEMENTATION FREEZE"
)


implementation_record = {
    "schema":
        "stage26_0d_preflight_implementation_v1",

    "stage":
        26,

    "parent_protocol_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "measurement_seed":
        MEASUREMENT_SEED,

    "group_a_seed_derivation":
        "derived_seed = 26042 + batch_size",

    "group_b_seed_derivation":
        "derived_seed = 26042 + 1000000 + batch_size",

    "group_a_tree_input":
        (
            "float32(scaler.mean_ + Z * scaler.scale_)"
        ),

    "group_a_ft_input":
        "Z float32",

    "group_b_cnn_boundary":
        (
            "uint8 image + bool mask; frozen CNN forward performs "
            "float32 conversion and divide-by-255 internally"
        ),

    "group_b_vit_boundary":
        (
            "float32 image/255 + bool mask before frozen ViT forward"
        ),

    "preflight_batch_size":
        1,

    "preflight_prediction_per_condition":
        1,

    "timing_allowed":
        False,

    "memory_profiling_allowed":
        False,

    "holdout_access_allowed":
        False,

    "gpu_allowed":
        False,

    "worker_sha256":
        worker_sha,
}


IMPLEMENTATION_RECORD_PATH = (
    PREFLIGHT_LOCAL_DIR
    / "stage26_0d_preflight_implementation.json"
)


write_json(
    IMPLEMENTATION_RECORD_PATH,
    implementation_record,
)


implementation_sha = sha256_file(
    IMPLEMENTATION_RECORD_PATH
)


print("Implementation record:")
print(" ", IMPLEMENTATION_RECORD_PATH)

print("SHA256:")
print(" ", implementation_sha)


# =============================================================================
# 8. RUN 16 UNTIMERED PREFLIGHT SUBPROCESSES
# =============================================================================

banner(
    "STAGE26-0D :: RUN UNTIMERED CPU PREFLIGHT"
)


results = []

failures = []


condition_number = 0


for target in TARGETS:

    for (
        hardware_mode,
        mode,
    ) in CPU_MODES.items():

        condition_number += 1

        config = {
            "schema":
                "stage26_cpu_preflight_condition_v1",

            "repo":
                str(
                    REPO_DIR
                ),

            "target_id":
                target,

            "hardware_mode":
                hardware_mode,

            "thread_count":
                mode[
                    "threads"
                ],

            "affinity":
                mode[
                    "affinity"
                ],

            "batch_size":
                1,

            "measurement_seed":
                MEASUREMENT_SEED,

            "timing_allowed":
                False,

            "memory_profiling_allowed":
                False,
        }


        config_path = (
            PREFLIGHT_LOCAL_DIR
            / (
                f"preflight_{condition_number:02d}_"
                f"{target}_"
                f"{hardware_mode}.json"
            )
        )


        write_json(
            config_path,
            config,
        )


        env = os.environ.copy()


        thread_count = str(
            mode[
                "threads"
            ]
        )


        # Frozen thread variables passed BEFORE subprocess/framework imports.
        for key in [
            "OMP_NUM_THREADS",
            "MKL_NUM_THREADS",
            "OPENBLAS_NUM_THREADS",
            "NUMEXPR_NUM_THREADS",
            "BLIS_NUM_THREADS",
            "VECLIB_MAXIMUM_THREADS",
        ]:
            env[
                key
            ] = thread_count


        # Prevent accidental GPU path even if environment changes.
        env[
            "CUDA_VISIBLE_DEVICES"
        ] = ""


        p = subprocess.run(
            [
                sys.executable,
                str(
                    WORKER_PATH
                ),
                str(
                    config_path
                ),
            ],
            cwd=REPO_DIR,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=False,
        )


        if p.returncode != 0:

            failure = {
                "condition_number":
                    condition_number,

                "target_id":
                    target,

                "hardware_mode":
                    hardware_mode,

                "returncode":
                    p.returncode,

                "stdout":
                    p.stdout[-4000:],

                "stderr":
                    p.stderr[-8000:],
            }

            failures.append(
                failure
            )

            print(
                f"[FAIL] {condition_number:02d}/16 "
                f"{target:43s} "
                f"{hardware_mode}"
            )

            # Fail immediately. Do not continue discovering later
            # compatibility issues after an unresolved preflight failure.
            break


        lines = [
            line.strip()
            for line in p.stdout.splitlines()
            if line.strip()
        ]


        if not lines:
            raise RuntimeError(
                f"No worker JSON returned for {target}"
            )


        try:
            result = json.loads(
                lines[-1]
            )

        except Exception as exc:
            raise RuntimeError(
                "Unable to parse final worker output.\n\n"
                + p.stdout
                + "\n\nSTDERR:\n"
                + p.stderr
            ) from exc


        if result.get(
            "status"
        ) != "PASS":
            raise RuntimeError(
                f"Worker did not return PASS: {result}"
            )


        results.append(
            result
        )


        print(
            f"[PASS] {condition_number:02d}/16 "
            f"{target:43s} "
            f"{hardware_mode:21s} "
            f"output={result['output_sha256'][:16]}..."
        )


    if failures:
        break


# =============================================================================
# 9. FAILURE HANDLING
# =============================================================================

if failures:

    FAILURE_PATH = (
        PREFLIGHT_LOCAL_DIR
        / "stage26_0d_preflight_failure.json"
    )


    write_json(
        FAILURE_PATH,
        {
            "schema":
                "stage26_0d_preflight_failure_v1",

            "stage":
                26,

            "parent_protocol_commit":
                EXPECTED_HEAD,

            "timing_performed":
                False,

            "memory_profiling_performed":
                False,

            "holdout_accessed":
                False,

            "gpu_used":
                False,

            "failures":
                failures,
        },
    )


    print(
        "\nFailure receipt:"
    )

    print(
        " ",
        FAILURE_PATH,
    )


    first = failures[0]


    print(
        "\nSTDOUT:"
    )

    print(
        first[
            "stdout"
        ]
    )


    print(
        "\nSTDERR:"
    )

    print(
        first[
            "stderr"
        ]
    )


    raise RuntimeError(
        "STAGE26-0D PREFLIGHT FAILED. "
        "Do not begin benchmarking. "
        "Send the complete output for one scoped compatibility fix."
    )


# =============================================================================
# 10. CROSS-MODE DETERMINISM GATE
#
# Same model + same B=1 input should yield the same result fingerprint between
# the 1-core and 2-core CPU modes. This is NOT a performance comparison.
# =============================================================================

banner(
    "STAGE26-0D :: CROSS-MODE PREDICTION FINGERPRINT GATE"
)


by_target = {}


for result in results:

    by_target.setdefault(
        result[
            "target_id"
        ],
        {}
    )[
        result[
            "hardware_mode"
        ]
    ] = result


fingerprint_gate = []


for target in TARGETS:

    one = by_target[
        target
    ][
        "CPU_1_PHYSICAL_CORE"
    ]

    two = by_target[
        target
    ][
        "CPU_2_PHYSICAL_CORE"
    ]


    same = (
        one[
            "output_sha256"
        ]
        ==
        two[
            "output_sha256"
        ]
    )


    fingerprint_gate.append(
        {
            "target_id":
                target,

            "cpu1_sha256":
                one[
                    "output_sha256"
                ],

            "cpu2_sha256":
                two[
                    "output_sha256"
                ],

            "identical":
                same,
        }
    )


    print(
        f"{target:43s} "
        f"{'PASS' if same else 'FAIL'}"
    )


    if not same:
        raise RuntimeError(
            f"Prediction changed between CPU modes for {target}. "
            "Do not benchmark until resolved."
        )


# =============================================================================
# 11. PREFLIGHT RECEIPT
# =============================================================================

banner(
    "STAGE26-0D :: BUILD PREFLIGHT RECEIPT"
)


receipt = {
    "schema":
        "stage26_0d_cpu_preflight_receipt_v1",

    "stage":
        26,

    "status":
        "PASS",

    "completed_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_protocol_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "cpu_execution_plan_sha256":
        EXPECTED_PLAN_SHA256,

    "worker_sha256":
        worker_sha,

    "implementation_record_sha256":
        implementation_sha,

    "target_count":
        len(
            TARGETS
        ),

    "cpu_mode_count":
        len(
            CPU_MODES
        ),

    "preflight_condition_count":
        len(
            results
        ),

    "preflight_batch_size":
        1,

    "results":
        results,

    "cross_mode_fingerprint_gate":
        fingerprint_gate,

    "scientific_boundary_after_preflight": {
        "model_deserialization_permitted":
            True,

        "untimed_preflight_inference_performed":
            True,

        "performance_timing_performed":
            False,

        "latency_observations":
            0,

        "throughput_observations":
            0,

        "memory_measurements":
            0,

        "holdout_reopened":
            False,

        "gpu_used":
            False,

        "thresholds_changed":
            False,

        "models_changed":
            False,
    },

    "next_action":
        (
            "GIT_ANCHOR_STAGE26_0D_PREFLIGHT_BEFORE_FIRST_CPU_TIMING"
        ),
}


RECEIPT_PATH = (
    PREFLIGHT_LOCAL_DIR
    / "stage26_0d_cpu_preflight_receipt.json"
)


write_json(
    RECEIPT_PATH,
    receipt,
)


receipt_sha = sha256_file(
    RECEIPT_PATH
)


print("Preflight receipt:")
print(" ", RECEIPT_PATH)

print("SHA256:")
print(" ", receipt_sha)


# =============================================================================
# 12. COPY SUCCESSFUL PREFLIGHT PACKAGE INTO REPOSITORY
# =============================================================================

banner(
    "STAGE26-0D :: BUILD DURABLE PREFLIGHT PACKAGE"
)


if PREFLIGHT_REPO_DIR.exists():

    # This should not normally exist at this parent commit.
    if any(
        PREFLIGHT_REPO_DIR.iterdir()
    ):
        raise RuntimeError(
            "Stage26-0D repo package already exists unexpectedly."
        )

else:
    PREFLIGHT_REPO_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )


repo_worker = (
    PREFLIGHT_REPO_DIR
    / "stage26_cpu_preflight_worker.py"
)

repo_implementation = (
    PREFLIGHT_REPO_DIR
    / IMPLEMENTATION_RECORD_PATH.name
)

repo_receipt = (
    PREFLIGHT_REPO_DIR
    / RECEIPT_PATH.name
)


shutil.copy2(
    WORKER_PATH,
    repo_worker,
)

shutil.copy2(
    IMPLEMENTATION_RECORD_PATH,
    repo_implementation,
)

shutil.copy2(
    RECEIPT_PATH,
    repo_receipt,
)


# Package manifest.
PACKAGE_MANIFEST = (
    PREFLIGHT_REPO_DIR
    / "stage26_0d_preflight_package_manifest.json"
)


manifest_files = []


for path in sorted(
    PREFLIGHT_REPO_DIR.iterdir()
):

    if (
        path.is_file()
        and
        path.name
        !=
        PACKAGE_MANIFEST.name
    ):

        manifest_files.append(
            {
                "path":
                    str(
                        path.relative_to(
                            REPO_DIR
                        )
                    ),

                "size_bytes":
                    path.stat().st_size,

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )


package_manifest = {
    "schema":
        "stage26_0d_preflight_package_manifest_v1",

    "stage":
        26,

    "status":
        "READY_FOR_GIT_ANCHOR_BEFORE_FIRST_CPU_TIMING",

    "parent_protocol_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "preflight_receipt_sha256":
        receipt_sha,

    "files":
        manifest_files,

    "scientific_boundary": {
        "performance_timing_performed":
            False,

        "memory_profiled":
            False,

        "holdout_accessed":
            False,

        "gpu_used":
            False,
    },
}


write_json(
    PACKAGE_MANIFEST,
    package_manifest,
)


package_sha = sha256_file(
    PACKAGE_MANIFEST
)


print(
    "Repository package:"
)

print(
    " ",
    PREFLIGHT_REPO_DIR,
)

print(
    "\nPackage manifest SHA256:"
)

print(
    " ",
    package_sha,
)


# =============================================================================
# 13. FINAL GIT-STATE GATE
# =============================================================================

banner(
    "STAGE26-0D FINAL AUDIT"
)


STATUS_AFTER = git(
    "status",
    "--porcelain",
)


print(
    "Repository status:"
)

print(
    STATUS_AFTER
    if STATUS_AFTER
    else "<unexpected clean>"
)


if not STATUS_AFTER:
    raise RuntimeError(
        "Expected new Stage26-0D files."
    )


unexpected = []


for line in STATUS_AFTER.splitlines():

    path = line[3:]

    if not path.startswith(
        "results/stage26_deployment_profiling/stage26_0d_cpu_preflight/"
    ):
        unexpected.append(
            line
        )


if unexpected:
    raise RuntimeError(
        "Unexpected repository changes outside Stage26-0D:\n"
        + "\n".join(
            unexpected
        )
    )


# =============================================================================
# 14. CLOSURE
# =============================================================================

banner(
    "STAGE26-0D CPU PREFLIGHT COMPLETE"
)


print(
    "Protocol parent commit:"
)

print(
    " ",
    EXPECTED_HEAD,
)


print(
    "\nPreflight conditions:"
)

print(
    " ",
    len(
        results
    ),
    "/ 16 PASS",
)


print(
    "\nValidated targets:"
)

for target in TARGETS:
    print(
        "  PASS",
        target,
    )


print(
    "\nValidated CPU modes:"
)

print(
    "  PASS CPU_1_PHYSICAL_CORE affinity=[0] threads=1"
)

print(
    "  PASS CPU_2_PHYSICAL_CORE affinity=[0,1] threads=2"
)


print(
    "\nScientific state:"
)

print(
    "  MODEL DESERIALIZATION NOW VERIFIED"
)

print(
    "  EXACT FORWARD/PREDICT PATHS VERIFIED"
)

print(
    "  ONLY UNTIMERED BATCH-1 PREFLIGHT PREDICTIONS EXECUTED"
)

print(
    "  NO PERFORMANCE CLOCK USED"
)

print(
    "  LATENCY OBSERVATIONS = 0"
)

print(
    "  THROUGHPUT OBSERVATIONS = 0"
)

print(
    "  MEMORY MEASUREMENTS = 0"
)

print(
    "  HOLDOUT NOT REOPENED"
)

print(
    "  GPU NOT USED"
)


print(
    "\nPREFLIGHT RECEIPT SHA256:"
)

print(
    " ",
    receipt_sha,
)


print(
    "\nPACKAGE MANIFEST SHA256:"
)

print(
    " ",
    package_sha,
)


print(
    "\nCRITICAL NEXT ACTION:"
)

print(
    "  STAGE26-0D-GIT — COMMIT + PUSH + REMOTE VERIFY PREFLIGHT PACKAGE."
)

print(
    "  DO NOT START CPU TIMING UNTIL THAT REMOTE CHECK PASSES."
)


STAGE26-0D :: SCIENTIFIC / GIT GATE
Expected HEAD : f3957b964e659392c57d1b12881163288f2efedc
Local HEAD    : f3957b964e659392c57d1b12881163288f2efedc
origin/main   : f3957b964e659392c57d1b12881163288f2efedc
Repo clean    : True

measurement_protocol SHA: d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
execution_plan SHA      : b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363

STAGE26-0D :: SOURCE / ARTIFACT GATE
[OK] xgboost            results/stage16_classical_benchmark_checkpoint/stage16_3_tuned_models/XGBOOST_tuned.joblib
[OK] lightgbm           results/stage16_classical_benchmark_checkpoint/stage16_3_tuned_models/LIGHTGBM_tuned.joblib
[OK] catboost           results/stage16_classical_benchmark_checkpoint/stage16_3_tuned_models/CATBOOST_tuned.joblib
[OK] ft_seed7           results/stage15_transformer_checkpoint/stage15_4b_models/FT_BALANCED_seed_7_best_extended.pt
[OK] ft_seed29          results/stage15_transformer_checkpoint/stage15_4a_models/FT_BA

In [13]:
# =============================================================================
# STAGE26-0D-GIT — ANCHOR SUCCESSFUL CPU PREFLIGHT BEFORE FIRST TIMING
#
# PURPOSE
# -------
# Commit, push, and remotely verify the successful Stage26-0D CPU preflight
# package before any Stage26 performance timing begins.
#
# THIS CELL:
#   - verifies current parent protocol commit
#   - verifies exact preflight artifact hashes
#   - verifies only Stage26-0D files are dirty
#   - commits the preflight package
#   - pushes to origin/main
#   - fetches origin/main back
#   - verifies local == remote commit
#   - verifies exact remote file SHA256 values
#   - confirms repository clean
#
# THIS CELL DOES NOT:
#   - load any model
#   - execute inference
#   - perform timing
#   - perform memory profiling
#   - access holdout data
#   - use GPU
# =============================================================================

from __future__ import annotations

import os
import stat
import hashlib
import tempfile
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient


# =============================================================================
# CONFIG
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT_HEAD = (
    "f3957b964e659392c57d1b12881163288f2efedc"
)

PREFLIGHT_DIR_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0d_cpu_preflight"
)

WORKER_REL = (
    PREFLIGHT_DIR_REL
    + "/stage26_cpu_preflight_worker.py"
)

IMPLEMENTATION_REL = (
    PREFLIGHT_DIR_REL
    + "/stage26_0d_preflight_implementation.json"
)

RECEIPT_REL = (
    PREFLIGHT_DIR_REL
    + "/stage26_0d_cpu_preflight_receipt.json"
)

PACKAGE_REL = (
    PREFLIGHT_DIR_REL
    + "/stage26_0d_preflight_package_manifest.json"
)


EXPECTED_HASHES = {
    WORKER_REL:
        "248f7f3d1237cf1e8330f25f728ef4464cdbb6368f5a653ad205816e12190fb1",

    IMPLEMENTATION_REL:
        "b27634363c7b38599a0c4f090d772cb2293031d7de6c8037a25a800543b1fe77",

    RECEIPT_REL:
        "4c7fc55a0e44e53587d385989dfec4f3f57f20768c7dd4a2d08e2936d2ba21e5",

    PACKAGE_REL:
        "a1167be2211f9f9911c4307e36f8c9589e1f72580cd4278883ccfd23acaf727b",
}


EXPECTED_PROTOCOL_SHA = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

EXPECTED_EXECUTION_PLAN_SHA = (
    "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363"
)

COMMIT_MESSAGE = (
    "stage26: anchor successful CPU execution preflight"
)


# =============================================================================
# HELPERS
# =============================================================================

def banner(text: str):
    print("\n" + "=" * 104)
    print(text)
    print("=" * 104)


def run(
    cmd,
    *,
    env=None,
    check=True,
):
    p = subprocess.run(
        cmd,
        cwd=REPO_DIR,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "$ "
            + " ".join(cmd)
            + "\n\n"
            + p.stdout
        )

    return p


def git(
    *args,
    env=None,
    check=True,
):
    return run(
        ["git", *args],
        env=env,
        check=check,
    ).stdout.strip()


def sha256_file(path: Path):
    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def git_blob_bytes(
    ref: str,
    path: str,
):
    p = subprocess.run(
        [
            "git",
            "show",
            f"{ref}:{path}",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stderr.decode(
                "utf-8",
                errors="replace",
            )
        )

    return p.stdout


def sha256_bytes(data: bytes):
    return hashlib.sha256(
        data
    ).hexdigest()


# =============================================================================
# 1. PRE-COMMIT GIT GATE
# =============================================================================

banner(
    "STAGE26-0D-GIT :: PRE-COMMIT GATE"
)


HEAD_BEFORE = git(
    "rev-parse",
    "HEAD",
)

REMOTE_BEFORE = git(
    "rev-parse",
    "origin/main",
)

STATUS_BEFORE = git(
    "status",
    "--porcelain",
)


print("Expected parent :", EXPECTED_PARENT_HEAD)
print("Local HEAD      :", HEAD_BEFORE)
print("origin/main     :", REMOTE_BEFORE)

print("\nRepository status:")
print(
    STATUS_BEFORE
    if STATUS_BEFORE
    else "<CLEAN>"
)


if HEAD_BEFORE != EXPECTED_PARENT_HEAD:
    raise RuntimeError(
        "Unexpected local HEAD before Stage26-0D commit."
    )

if REMOTE_BEFORE != EXPECTED_PARENT_HEAD:
    raise RuntimeError(
        "origin/main changed after Stage26-0D preflight."
    )

if not STATUS_BEFORE:
    raise RuntimeError(
        "No uncommitted Stage26-0D package found."
    )


# Only Stage26-0D directory may be dirty.
unexpected = []

for line in STATUS_BEFORE.splitlines():

    path = line[3:]

    if not path.startswith(
        PREFLIGHT_DIR_REL + "/"
    ):
        unexpected.append(
            line
        )


if unexpected:
    raise RuntimeError(
        "Unexpected repository changes outside Stage26-0D:\n"
        + "\n".join(
            unexpected
        )
    )


print(
    "\n[PASS] Only the Stage26-0D preflight package is uncommitted."
)


# =============================================================================
# 2. EXACT LOCAL HASH GATE
# =============================================================================

banner(
    "STAGE26-0D-GIT :: LOCAL HASH GATE"
)


for relpath, expected_sha in EXPECTED_HASHES.items():

    path = (
        REPO_DIR
        / relpath
    )

    if not path.exists():
        raise FileNotFoundError(
            path
        )

    actual_sha = sha256_file(
        path
    )

    ok = (
        actual_sha
        ==
        expected_sha
    )

    print(
        f"{Path(relpath).name:48s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual_sha}"
    )

    if not ok:
        raise RuntimeError(
            f"Preflight artifact changed before commit: {relpath}"
        )


# =============================================================================
# 3. CONFIRM PREFLIGHT RECEIPT SCIENTIFIC CONTENT
# =============================================================================

banner(
    "STAGE26-0D-GIT :: PREFLIGHT CONTENT GATE"
)


import json


receipt_path = (
    REPO_DIR
    / RECEIPT_REL
)

receipt = json.loads(
    receipt_path.read_text(
        encoding="utf-8"
    )
)


print(
    "Status               :",
    receipt["status"],
)

print(
    "Conditions           :",
    receipt["preflight_condition_count"],
)

boundary = receipt[
    "scientific_boundary_after_preflight"
]


for key, value in boundary.items():
    print(
        f"{key:38s}: {value}"
    )


if receipt["status"] != "PASS":
    raise RuntimeError(
        "Preflight receipt is not PASS."
    )

if receipt[
    "preflight_condition_count"
] != 16:
    raise RuntimeError(
        "Expected exactly 16 successful preflight conditions."
    )


required_boundary = {
    "performance_timing_performed":
        False,

    "latency_observations":
        0,

    "throughput_observations":
        0,

    "memory_measurements":
        0,

    "holdout_reopened":
        False,

    "gpu_used":
        False,

    "thresholds_changed":
        False,

    "models_changed":
        False,
}


for key, expected in required_boundary.items():

    actual = boundary.get(
        key
    )

    if actual != expected:
        raise RuntimeError(
            f"Scientific boundary mismatch: "
            f"{key}={actual!r}, expected {expected!r}"
        )


# Verify all 16 condition results are PASS.
result_rows = receipt[
    "results"
]

if len(result_rows) != 16:
    raise RuntimeError(
        "Preflight receipt does not contain exactly 16 result rows."
    )


for row in result_rows:

    if row.get(
        "status"
    ) != "PASS":
        raise RuntimeError(
            "Non-PASS preflight condition found."
        )

    if row.get(
        "timing_performed"
    ) is not False:
        raise RuntimeError(
            "A preflight worker reports timing."
        )

    if row.get(
        "memory_profiling_performed"
    ) is not False:
        raise RuntimeError(
            "A preflight worker reports memory profiling."
        )

    if row.get(
        "holdout_accessed"
    ) is not False:
        raise RuntimeError(
            "A preflight worker reports holdout access."
        )

    if row.get(
        "gpu_used"
    ) is not False:
        raise RuntimeError(
            "A preflight worker reports GPU use."
        )


# Verify 8/8 cross-mode fingerprint gates.
fingerprints = receipt[
    "cross_mode_fingerprint_gate"
]

if len(fingerprints) != 8:
    raise RuntimeError(
        "Expected 8 cross-mode fingerprint rows."
    )


for row in fingerprints:

    if row.get(
        "identical"
    ) is not True:
        raise RuntimeError(
            f"Cross-mode fingerprint mismatch: "
            f"{row['target_id']}"
        )


print(
    "\n[PASS] 16/16 conditions valid."
)

print(
    "[PASS] 8/8 cross-mode prediction fingerprints identical."
)

print(
    "[PASS] No timing/memory/holdout/GPU observations recorded."
)


# =============================================================================
# 4. STAGE EXACT PREFLIGHT DIRECTORY
# =============================================================================

banner(
    "STAGE26-0D-GIT :: STAGE FILES"
)


git(
    "add",
    "--",
    PREFLIGHT_DIR_REL,
)


STAGED = git(
    "diff",
    "--cached",
    "--name-status",
)


print(STAGED)


if not STAGED:
    raise RuntimeError(
        "No files staged."
    )


for line in STAGED.splitlines():

    path = line.split(
        "\t"
    )[-1]

    if not path.startswith(
        PREFLIGHT_DIR_REL + "/"
    ):
        raise RuntimeError(
            "Unexpected staged path:\n"
            + line
        )


# =============================================================================
# 5. GIT IDENTITY
# =============================================================================

name = git(
    "config",
    "--get",
    "user.name",
    check=False,
)

email = git(
    "config",
    "--get",
    "user.email",
    check=False,
)


if not name:

    git(
        "config",
        "user.name",
        "themubasshir",
    )


if not email:

    git(
        "config",
        "user.email",
        "themubasshir@users.noreply.github.com",
    )


# =============================================================================
# 6. COMMIT
# =============================================================================

banner(
    "STAGE26-0D-GIT :: COMMIT"
)


commit_output = git(
    "commit",
    "-m",
    COMMIT_MESSAGE,
)


print(
    commit_output
)


NEW_HEAD = git(
    "rev-parse",
    "HEAD",
)

NEW_PARENT = git(
    "rev-parse",
    "HEAD^",
)


print(
    "\nParent commit  :",
    NEW_PARENT,
)

print(
    "Preflight commit:",
    NEW_HEAD,
)


if NEW_PARENT != EXPECTED_PARENT_HEAD:
    raise RuntimeError(
        "Stage26-0D commit has wrong parent."
    )


# =============================================================================
# 7. AUTHENTICATION
# =============================================================================

banner(
    "STAGE26-0D-GIT :: GITHUB AUTH"
)


secret_client = UserSecretsClient()


SECRET_NAMES = [
    "GITHUB_TOKEN",
    "github_token",
    "GH_TOKEN",
    "gh_token",
    "GITHUB_PAT",
    "github_pat",
    "GH_PAT",
    "gh_pat",
]


token = None
token_name = None


for secret_name in SECRET_NAMES:

    try:
        value = secret_client.get_secret(
            secret_name
        )

    except Exception:
        value = None

    if value:

        token = value.strip()
        token_name = secret_name

        break


if not token:
    raise RuntimeError(
        "No usable GitHub token found in Kaggle Secrets."
    )


print(
    f"[FOUND] {token_name} "
    f"({len(token)} characters)"
)

print(
    "Token value will not be printed."
)


# =============================================================================
# 8. PUSH
# =============================================================================

banner(
    "STAGE26-0D-GIT :: PUSH"
)


fd, askpass_name = tempfile.mkstemp(
    prefix="stage26_0d_askpass_",
    suffix=".sh",
)

os.close(
    fd
)

askpass = Path(
    askpass_name
)


askpass.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' "x-access-token" ;;
  *Password*) printf '%s\\n' "$GITHUB_TOKEN" ;;
  *)          printf '%s\\n' "" ;;
esac
""",
    encoding="utf-8",
)


askpass.chmod(
    stat.S_IRUSR
    |
    stat.S_IWUSR
    |
    stat.S_IXUSR
)


push_env = os.environ.copy()

push_env[
    "GITHUB_TOKEN"
] = token

push_env[
    "GIT_ASKPASS"
] = str(
    askpass
)

push_env[
    "GIT_TERMINAL_PROMPT"
] = "0"


try:

    push_result = run(
        [
            "git",
            "push",
            "origin",
            "HEAD:main",
        ],
        env=push_env,
        check=True,
    )

    print(
        push_result.stdout
    )

finally:

    try:
        askpass.unlink()
    except FileNotFoundError:
        pass

    token = None


# =============================================================================
# 9. FETCH REMOTE BACK
# =============================================================================

banner(
    "STAGE26-0D-GIT :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


LOCAL_HEAD = git(
    "rev-parse",
    "HEAD",
)

REMOTE_HEAD = git(
    "rev-parse",
    "origin/main",
)


print(
    "Local HEAD :",
    LOCAL_HEAD,
)

print(
    "Remote HEAD:",
    REMOTE_HEAD,
)


if LOCAL_HEAD != NEW_HEAD:
    raise RuntimeError(
        "Local HEAD unexpectedly changed."
    )

if REMOTE_HEAD != NEW_HEAD:
    raise RuntimeError(
        "origin/main does not contain the Stage26-0D commit."
    )


ls_remote = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


LS_REMOTE_SHA = (
    ls_remote.split()[0]
    if ls_remote
    else None
)


print(
    "ls-remote :",
    LS_REMOTE_SHA,
)


if LS_REMOTE_SHA != NEW_HEAD:
    raise RuntimeError(
        "Independent remote ref verification failed."
    )


# =============================================================================
# 10. VERIFY REMOTE FILE BYTES
# =============================================================================

banner(
    "STAGE26-0D-GIT :: REMOTE SHA GATE"
)


for relpath, expected_sha in EXPECTED_HASHES.items():

    remote_bytes = git_blob_bytes(
        "origin/main",
        relpath,
    )

    remote_sha = sha256_bytes(
        remote_bytes
    )

    ok = (
        remote_sha
        ==
        expected_sha
    )

    print(
        f"{Path(relpath).name:48s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{remote_sha}"
    )

    if not ok:
        raise RuntimeError(
            f"Remote SHA mismatch: {relpath}"
        )


# =============================================================================
# 11. REVERIFY ORIGINAL MEASUREMENT LOCK REMOTELY
# =============================================================================

banner(
    "STAGE26-0D-GIT :: ORIGINAL PROTOCOL STILL IMMUTABLE"
)


PROTOCOL_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/measurement_protocol.json"
)

PLAN_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/cpu_execution_plan.json"
)


remote_protocol_sha = sha256_bytes(
    git_blob_bytes(
        "origin/main",
        PROTOCOL_REL,
    )
)

remote_plan_sha = sha256_bytes(
    git_blob_bytes(
        "origin/main",
        PLAN_REL,
    )
)


print(
    "measurement_protocol:",
    remote_protocol_sha,
)

print(
    "cpu_execution_plan  :",
    remote_plan_sha,
)


if remote_protocol_sha != EXPECTED_PROTOCOL_SHA:
    raise RuntimeError(
        "Original Stage26 measurement protocol changed."
    )

if remote_plan_sha != EXPECTED_EXECUTION_PLAN_SHA:
    raise RuntimeError(
        "Original Stage26 execution plan changed."
    )


print(
    "\n[PASS] Original measurement lock remains byte-identical."
)


# =============================================================================
# 12. FINAL REPOSITORY AUDIT
# =============================================================================

banner(
    "STAGE26-0D-GIT :: FINAL AUDIT"
)


FINAL_STATUS = git(
    "status",
    "--porcelain",
)


print(
    "Repository clean:",
    FINAL_STATUS == "",
)


if FINAL_STATUS:

    print(
        FINAL_STATUS
    )

    raise RuntimeError(
        "Repository is dirty after Stage26-0D push."
    )


# =============================================================================
# 13. CLOSURE
# =============================================================================

banner(
    "STAGE26-0D-GIT COMPLETE"
)


print(
    "PROTOCOL PARENT:"
)

print(
    " ",
    EXPECTED_PARENT_HEAD,
)


print(
    "\nSTAGE26-0D PREFLIGHT COMMIT:"
)

print(
    " ",
    NEW_HEAD,
)


print(
    "\nREMOTE MAIN:"
)

print(
    " ",
    REMOTE_HEAD,
)


print(
    "\nPREFLIGHT RECEIPT SHA256:"
)

print(
    " ",
    EXPECTED_HASHES[
        RECEIPT_REL
    ],
)


print(
    "\nPREFLIGHT PACKAGE SHA256:"
)

print(
    " ",
    EXPECTED_HASHES[
        PACKAGE_REL
    ],
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  COMMIT MATCH                 : PASS"
)

print(
    "  LS-REMOTE                    : PASS"
)

print(
    "  WORKER SHA                   : PASS"
)

print(
    "  IMPLEMENTATION SHA           : PASS"
)

print(
    "  PREFLIGHT RECEIPT SHA        : PASS"
)

print(
    "  PREFLIGHT PACKAGE SHA        : PASS"
)

print(
    "  ORIGINAL PROTOCOL IMMUTABLE  : PASS"
)

print(
    "  ORIGINAL EXECUTION PLAN      : PASS"
)

print(
    "  REPOSITORY CLEAN             : PASS"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  STAGE26 MEASUREMENT PROTOCOL REMOTELY FROZEN"
)

print(
    "  STAGE26 CPU EXECUTION PREFLIGHT REMOTELY FROZEN"
)

print(
    "  16 / 16 UNTIMERED EXECUTION CONDITIONS PASS"
)

print(
    "  8 / 8 CROSS-MODE PREDICTION FINGERPRINTS PASS"
)

print(
    "  LATENCY OBSERVATIONS = 0"
)

print(
    "  THROUGHPUT OBSERVATIONS = 0"
)

print(
    "  MEMORY MEASUREMENTS = 0"
)

print(
    "  HOLDOUT NOT REOPENED"
)

print(
    "  GPU NOT USED"
)


print(
    "\nNEXT:"
)

print(
    "  STAGE26-1 — CPU COLD-START PROFILING"
)

print(
    "  The first Stage26 performance clock becomes permitted."
)

print(
    "  All cold-start runs will use fresh interpreter subprocesses"
)

print(
    "  under the already-frozen CPU_1_PHYSICAL_CORE condition."
)


STAGE26-0D-GIT :: PRE-COMMIT GATE
Expected parent : f3957b964e659392c57d1b12881163288f2efedc
Local HEAD      : f3957b964e659392c57d1b12881163288f2efedc
origin/main     : f3957b964e659392c57d1b12881163288f2efedc

Repository status:
?? results/stage26_deployment_profiling/stage26_0d_cpu_preflight/

[PASS] Only the Stage26-0D preflight package is uncommitted.

STAGE26-0D-GIT :: LOCAL HASH GATE
stage26_cpu_preflight_worker.py                  PASS 248f7f3d1237cf1e8330f25f728ef4464cdbb6368f5a653ad205816e12190fb1
stage26_0d_preflight_implementation.json         PASS b27634363c7b38599a0c4f090d772cb2293031d7de6c8037a25a800543b1fe77
stage26_0d_cpu_preflight_receipt.json            PASS 4c7fc55a0e44e53587d385989dfec4f3f57f20768c7dd4a2d08e2936d2ba21e5
stage26_0d_preflight_package_manifest.json       PASS a1167be2211f9f9911c4307e36f8c9589e1f72580cd4278883ccfd23acaf727b

STAGE26-0D-GIT :: PREFLIGHT CONTENT GATE
Status               : PASS
Conditions           : 16
gpu_used                         

In [14]:
# =============================================================================
# STAGE26-1 — CPU COLD-START PROFILING
#
# FIRST PERFORMANCE-MEASUREMENT CHECKPOINT IN STAGE26
#
# FROZEN PARENT:
#   c4502167547b7835ef4fa18ab9b1103b671e6894
#
# PURPOSE
# -------
# Measure cold-start behavior for all 8 frozen deployment targets under the
# primary frozen CPU condition:
#
#   CPU_1_PHYSICAL_CORE
#   affinity = [0]
#   threads  = 1
#   batch    = 1
#
# 20 fresh Python interpreter processes per target.
# 160 total cold-start repetitions.
#
# PRIMARY RAW COMPONENTS
# ----------------------
#   process_spawn_to_worker_ready_ns
#   framework_import_ns
#   model_deserialization_load_ns
#   first_prediction_ns
#   spawn_to_first_output_ns
#
# SUPPLEMENTARY DIAGNOSTICS
# -------------------------
#   input_preparation_ns
#   parent_process_total_ns
#
# SCIENTIFIC RULES
# ----------------
#   - fresh interpreter every repetition
#   - no fork reuse
#   - deterministic randomized execution order
#   - exact CPU affinity/thread freeze
#   - deterministic B=1 synthetic input
#   - no holdout access
#   - no memory profiling
#   - no GPU
#   - raw nanosecond observations retained
#   - no post-hoc iteration changes
#   - environment gate before every repetition
#
# IMPORTANT
# ---------
# This cell is intentionally NOT silently rerunnable after measurements begin.
# If partial Stage26-1 raw results already exist, it aborts.
# =============================================================================

from __future__ import annotations

import os
import sys
import csv
import json
import time
import stat
import shutil
import hashlib
import subprocess
import textwrap
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import psutil


# =============================================================================
# 0. FROZEN CONSTANTS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

WORKERS_DIR = (
    STAGE26_ROOT
    / "workers"
)

COLD_DIR = (
    STAGE26_ROOT
    / "cold_start"
)

REPO_COLD_DIR = (
    REPO_DIR
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_1_cpu_cold_start"
)

WORKERS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

COLD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


EXPECTED_HEAD = (
    "c4502167547b7835ef4fa18ab9b1103b671e6894"
)

EXPECTED_PROTOCOL_SHA = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

EXPECTED_PLAN_SHA = (
    "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363"
)

EXPECTED_PREFLIGHT_RECEIPT_SHA = (
    "4c7fc55a0e44e53587d385989dfec4f3f57f20768c7dd4a2d08e2936d2ba21e5"
)

MEASUREMENT_SEED = 26042

COLD_REPETITIONS = 20

PRIMARY_AFFINITY = [0]

PRIMARY_THREADS = 1

BATCH_SIZE = 1

CPU_UTILIZATION_GATE_PERCENT = 20.0

MIN_AVAILABLE_RAM_GIB = 8.0

ENVIRONMENT_RETRY_COUNT = 3

ENVIRONMENT_RETRY_COOLDOWN_SECONDS = 5

CPU_UTILIZATION_SAMPLE_SECONDS = 1.0

CONDITION_COOLDOWN_SECONDS = 2

CONDITION_TIMEOUT_SECONDS = 600

BOOTSTRAP_REPLICATES = 2000

BOOTSTRAP_SEED = 26042


TARGETS = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
    "ENS_LGBM_XGB_EQUAL",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
]


# =============================================================================
# 1. OUTPUT PATHS
# =============================================================================

WORKER_PATH = (
    WORKERS_DIR
    / "stage26_cold_start_worker.py"
)

SCHEDULE_PATH = (
    COLD_DIR
    / "stage26_1_cold_start_schedule.json"
)

IMPLEMENTATION_PATH = (
    COLD_DIR
    / "stage26_1_cold_start_implementation.json"
)

RAW_JSONL = (
    COLD_DIR
    / "stage26_1_cold_start_raw.jsonl"
)

RAW_CSV = (
    COLD_DIR
    / "stage26_1_cold_start_raw.csv"
)

SUMMARY_CSV = (
    COLD_DIR
    / "stage26_1_cold_start_summary.csv"
)

SUMMARY_JSON = (
    COLD_DIR
    / "stage26_1_cold_start_summary.json"
)

RECEIPT_PATH = (
    COLD_DIR
    / "stage26_1_cold_start_receipt.json"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def banner(text: str):
    print("\n" + "=" * 108)
    print(text)
    print("=" * 108)


def git(*args):
    p = subprocess.run(
        ["git", *args],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(path: Path):
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            b = f.read(
                8 * 1024 * 1024
            )

            if not b:
                break

            h.update(b)

    return h.hexdigest()


def write_json(path: Path, obj):
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )


def stable_seed(text: str):
    digest = hashlib.sha256(
        text.encode("utf-8")
    ).digest()

    return int.from_bytes(
        digest[:8],
        "little",
    ) % (2**32)


def percentile(values, q):
    return float(
        np.percentile(
            np.asarray(
                values,
                dtype=np.float64,
            ),
            q,
        )
    )


def bootstrap_ci(
    values,
    statistic,
    *,
    seed,
    replicates=BOOTSTRAP_REPLICATES,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    rng = np.random.default_rng(
        int(seed)
    )

    n = len(values)

    estimates = np.empty(
        replicates,
        dtype=np.float64,
    )

    for i in range(
        replicates
    ):
        sample = values[
            rng.integers(
                0,
                n,
                size=n,
            )
        ]

        estimates[i] = statistic(
            sample
        )

    return (
        float(
            np.percentile(
                estimates,
                2.5,
            )
        ),
        float(
            np.percentile(
                estimates,
                97.5,
            )
        ),
    )


# =============================================================================
# 3. GIT / SCIENTIFIC PARENT GATE
# =============================================================================

banner(
    "STAGE26-1 :: PARENT / PROTOCOL GATE"
)


HEAD = git(
    "rev-parse",
    "HEAD",
)

REMOTE = git(
    "rev-parse",
    "origin/main",
)

STATUS = git(
    "status",
    "--porcelain",
)


print("Expected HEAD :", EXPECTED_HEAD)
print("Local HEAD    :", HEAD)
print("origin/main   :", REMOTE)
print("Repository clean:", STATUS == "")


if HEAD != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected local HEAD before Stage26-1."
    )

if REMOTE != EXPECTED_HEAD:
    raise RuntimeError(
        "origin/main does not match frozen Stage26-0D preflight commit."
    )

if STATUS:
    raise RuntimeError(
        "Repository must be clean before first performance measurement:\n"
        + STATUS
    )


PROTOCOL_PATH = (
    REPO_DIR
    / "results/stage26_deployment_profiling/"
      "stage26_0_protocol_lock/"
      "measurement_protocol.json"
)

PLAN_PATH = (
    REPO_DIR
    / "results/stage26_deployment_profiling/"
      "stage26_0_protocol_lock/"
      "cpu_execution_plan.json"
)

PREFLIGHT_RECEIPT = (
    REPO_DIR
    / "results/stage26_deployment_profiling/"
      "stage26_0d_cpu_preflight/"
      "stage26_0d_cpu_preflight_receipt.json"
)


for path in [
    PROTOCOL_PATH,
    PLAN_PATH,
    PREFLIGHT_RECEIPT,
]:
    if not path.exists():
        raise FileNotFoundError(
            path
        )


actual_protocol_sha = sha256_file(
    PROTOCOL_PATH
)

actual_plan_sha = sha256_file(
    PLAN_PATH
)

actual_preflight_sha = sha256_file(
    PREFLIGHT_RECEIPT
)


print(
    "\nProtocol SHA :",
    actual_protocol_sha,
)

print(
    "Plan SHA     :",
    actual_plan_sha,
)

print(
    "Preflight SHA:",
    actual_preflight_sha,
)


if actual_protocol_sha != EXPECTED_PROTOCOL_SHA:
    raise RuntimeError(
        "Measurement protocol hash mismatch."
    )

if actual_plan_sha != EXPECTED_PLAN_SHA:
    raise RuntimeError(
        "CPU execution-plan hash mismatch."
    )

if actual_preflight_sha != EXPECTED_PREFLIGHT_RECEIPT_SHA:
    raise RuntimeError(
        "CPU preflight receipt hash mismatch."
    )


# =============================================================================
# 4. ANTI-DOUBLE-RUN GATE
# =============================================================================

banner(
    "STAGE26-1 :: ANTI-DOUBLE-RUN GATE"
)


existing_measurement_artifacts = [
    path
    for path in [
        RAW_JSONL,
        RAW_CSV,
        SUMMARY_CSV,
        SUMMARY_JSON,
        RECEIPT_PATH,
    ]
    if path.exists()
]


if existing_measurement_artifacts:

    print(
        "Existing Stage26-1 measurement artifacts:"
    )

    for path in existing_measurement_artifacts:
        print(
            " ",
            path,
        )

    raise RuntimeError(
        "Stage26-1 cold-start measurements already began or completed. "
        "Do NOT silently rerun this cell."
    )


if REPO_COLD_DIR.exists():
    raise RuntimeError(
        "A Stage26-1 repository package already exists unexpectedly."
    )


print(
    "[PASS] No previous Stage26-1 cold-start observations exist."
)


# =============================================================================
# 5. LOAD EXPECTED PREFLIGHT FINGERPRINTS
# =============================================================================

banner(
    "STAGE26-1 :: PREDICTION-INTEGRITY REFERENCE"
)


preflight = json.loads(
    PREFLIGHT_RECEIPT.read_text(
        encoding="utf-8"
    )
)


expected_fingerprints = {}


for row in preflight[
    "results"
]:

    if (
        row["hardware_mode"]
        ==
        "CPU_1_PHYSICAL_CORE"
    ):
        expected_fingerprints[
            row["target_id"]
        ] = row[
            "output_sha256"
        ]


if set(
    expected_fingerprints
) != set(
    TARGETS
):
    raise RuntimeError(
        "Unable to resolve all 8 expected preflight fingerprints."
    )


for target in TARGETS:
    print(
        f"{target:43s} "
        f"{expected_fingerprints[target][:20]}..."
    )


# =============================================================================
# 6. COLD-START WORKER
# =============================================================================

banner(
    "STAGE26-1 :: CREATE COLD-START WORKER"
)


worker_source = r'''
from __future__ import annotations

import os
import sys
import json
import hashlib
import importlib.util
from pathlib import Path
from time import perf_counter_ns


# Earliest practical timestamp after interpreter + stdlib startup.
PROCESS_ENTRY_NS = perf_counter_ns()


def import_path(name, path):

    spec = importlib.util.spec_from_file_location(
        name,
        str(path),
    )

    if spec is None or spec.loader is None:
        raise RuntimeError(
            f"Unable to import {path}"
        )

    module = importlib.util.module_from_spec(
        spec
    )

    spec.loader.exec_module(
        module
    )

    return module


def sha256_array(array):

    import numpy as np

    a = np.ascontiguousarray(
        array
    )

    h = hashlib.sha256()

    h.update(
        str(
            a.dtype
        ).encode(
            "ascii"
        )
    )

    h.update(
        json.dumps(
            list(
                a.shape
            )
        ).encode(
            "ascii"
        )
    )

    h.update(
        a.tobytes(
            order="C"
        )
    )

    return h.hexdigest()


def extract_torch_state(obj):

    import torch

    if not isinstance(
        obj,
        dict,
    ):
        raise TypeError(
            f"Unsupported checkpoint root type: {type(obj)}"
        )

    if obj and all(
        torch.is_tensor(v)
        for v in obj.values()
    ):
        return obj

    for key in [
        "model_state_dict",
        "state_dict",
        "model_state",
        "network_state_dict",
        "model",
    ]:

        candidate = obj.get(
            key
        )

        if (
            isinstance(
                candidate,
                dict,
            )
            and candidate
            and all(
                torch.is_tensor(v)
                for v in candidate.values()
            )
        ):
            return candidate

    raise RuntimeError(
        "Unable to identify state_dict."
    )


def torch_load_state(path):

    import torch

    try:
        obj = torch.load(
            str(path),
            map_location="cpu",
            weights_only=True,
        )

    except TypeError:
        obj = torch.load(
            str(path),
            map_location="cpu",
        )

    return extract_torch_state(
        obj
    )


def configure_torch_threads(
    thread_count,
):

    import torch

    torch.set_num_threads(
        int(
            thread_count
        )
    )

    try:
        torch.set_num_interop_threads(
            1
        )
    except RuntimeError:
        pass

    if int(
        torch.get_num_threads()
    ) != int(
        thread_count
    ):
        raise RuntimeError(
            "PyTorch intra-op thread mismatch."
        )

    if int(
        torch.get_num_interop_threads()
    ) != 1:
        raise RuntimeError(
            "PyTorch inter-op thread mismatch."
        )


def build_group_a_input(
    *,
    repo,
    seed,
):

    import joblib
    import numpy as np

    scaler = joblib.load(
        repo
        / "results/stage15_transformer_checkpoint/"
          "stage15_2_standard_scaler.joblib"
    )

    if int(
        scaler.n_features_in_
    ) != 70:
        raise RuntimeError(
            "Scaler feature count mismatch."
        )

    derived_seed = (
        int(seed)
        +
        1
    )

    rng = np.random.default_rng(
        derived_seed
    )

    Z = rng.normal(
        0.0,
        1.0,
        size=(
            1,
            70,
        ),
    ).astype(
        np.float32
    )

    mean = np.asarray(
        scaler.mean_,
        dtype=np.float32,
    )

    scale = np.asarray(
        scaler.scale_,
        dtype=np.float32,
    )

    X_raw = (
        mean[None, :]
        +
        Z
        *
        scale[None, :]
    ).astype(
        np.float32,
        copy=False,
    )

    return Z, X_raw


def build_ft_input(
    *,
    seed,
):

    import numpy as np

    derived_seed = (
        int(seed)
        +
        1
    )

    rng = np.random.default_rng(
        derived_seed
    )

    return rng.normal(
        0.0,
        1.0,
        size=(
            1,
            70,
        ),
    ).astype(
        np.float32
    )


def make_ipv4_packet(
    rng,
    protocol,
    length,
):

    import numpy as np

    packet = bytearray(
        rng.integers(
            0,
            256,
            size=int(length),
            dtype=np.uint8,
        ).tobytes()
    )

    packet[0] = 0x45

    packet[2:4] = int(
        length
    ).to_bytes(
        2,
        "big",
    )

    packet[6] = (
        packet[6]
        &
        0xE0
    )

    packet[7] = 0

    packet[9] = int(
        protocol
    )

    return bytes(
        packet
    )


def build_packet_input(
    *,
    encoder,
    seed,
):

    import numpy as np

    derived_seed = (
        int(seed)
        +
        1_000_001
    )

    rng = np.random.default_rng(
        derived_seed
    )

    packet_count = int(
        rng.integers(
            1,
            65,
        )
    )

    packets = []

    for _ in range(
        packet_count
    ):

        protocol = (
            6
            if int(
                rng.integers(
                    0,
                    2,
                )
            ) == 0
            else 17
        )

        packet_length = int(
            rng.integers(
                40,
                257,
            )
        )

        packets.append(
            make_ipv4_packet(
                rng,
                protocol,
                packet_length,
            )
        )

    image, mask = encoder.encode_flow(
        packets
    )

    image_uint8 = image[
        None,
        None,
        :,
        :,
    ]

    mask = mask[
        None,
        None,
        :,
        :,
    ]

    image_scaled = (
        image_uint8.astype(
            np.float32
        )
        /
        np.float32(
            255.0
        )
    )

    return (
        image_uint8,
        image_scaled,
        mask,
    )


def validate_probability(probability):

    import numpy as np

    p = np.asarray(
        probability
    ).reshape(
        -1
    )

    if p.shape != (
        1,
    ):
        raise RuntimeError(
            f"Unexpected output shape: {p.shape}"
        )

    if not np.isfinite(
        p
    ).all():
        raise RuntimeError(
            "Non-finite prediction."
        )

    if (
        (p < 0.0).any()
        or
        (p > 1.0).any()
    ):
        raise RuntimeError(
            "Invalid probability."
        )

    return p


def build_ft_model(
    *,
    ft_module,
    arch_record,
    checkpoint,
):

    import torch

    arch = arch_record[
        "architecture"
    ]

    model = ft_module.NumericFTTransformer(
        n_features=int(
            arch_record[
                "input_predictor_count"
            ]
        ),
        d_token=int(
            arch[
                "d_token"
            ]
        ),
        n_heads=int(
            arch[
                "n_heads"
            ]
        ),
        n_layers=int(
            arch[
                "n_layers"
            ]
        ),
        d_ff=int(
            arch[
                "d_ff"
            ]
        ),
        dropout=float(
            arch[
                "dropout"
            ]
        ),
    )

    state = torch_load_state(
        checkpoint
    )

    model.load_state_dict(
        state,
        strict=True,
    )

    model.eval()

    parameter_count = sum(
        int(
            p.numel()
        )
        for p in model.parameters()
        if p.requires_grad
    )

    if parameter_count != 159169:
        raise RuntimeError(
            f"FT parameter mismatch: {parameter_count}"
        )

    return model


def main():

    config = json.loads(
        Path(
            sys.argv[1]
        ).read_text(
            encoding="utf-8"
        )
    )

    repo = Path(
        config[
            "repo"
        ]
    )

    target = config[
        "target_id"
    ]

    thread_count = int(
        config[
            "thread_count"
        ]
    )

    affinity = [
        int(x)
        for x in config[
            "affinity"
        ]
    ]

    seed = int(
        config[
            "measurement_seed"
        ]
    )

    parent_spawn_start_ns = int(
        os.environ[
            "STAGE26_PARENT_SPAWN_START_NS"
        ]
    )


    # -------------------------------------------------------------------------
    # Exact CPU placement before ML imports.
    # -------------------------------------------------------------------------

    if hasattr(
        os,
        "sched_setaffinity",
    ):
        os.sched_setaffinity(
            0,
            set(
                affinity
            ),
        )

    observed_affinity = (
        sorted(
            os.sched_getaffinity(
                0
            )
        )
        if hasattr(
            os,
            "sched_getaffinity",
        )
        else None
    )

    if (
        observed_affinity is not None
        and
        observed_affinity != affinity
    ):
        raise RuntimeError(
            f"Affinity mismatch: "
            f"{observed_affinity} != {affinity}"
        )


    worker_ready_ns = perf_counter_ns()


    # -------------------------------------------------------------------------
    # FRAMEWORK IMPORT COMPONENT
    # -------------------------------------------------------------------------

    framework_import_start_ns = (
        perf_counter_ns()
    )


    if target in {
        "STAGE16_XGBOOST_TUNED",
    }:

        import numpy as np
        import joblib
        import xgboost


    elif target in {
        "STAGE16_LIGHTGBM_TUNED",
    }:

        import numpy as np
        import joblib
        import lightgbm


    elif target in {
        "STAGE16_CATBOOST_TUNED",
    }:

        import numpy as np
        import joblib
        import catboost


    elif target == "ENS_LGBM_XGB_EQUAL":

        import numpy as np
        import joblib
        import xgboost
        import lightgbm


    elif target in {
        "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
        "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
    }:

        import numpy as np
        import torch

        configure_torch_threads(
            thread_count
        )

        ft_module = import_path(
            "stage26_ft_module",
            (
                repo
                / "results/stage15_transformer_checkpoint/"
                  "ft_transformer_numeric.py"
            ),
        )


    elif target == "STAGE20_MASKED_CNN_V1":

        import numpy as np
        import torch

        configure_torch_threads(
            thread_count
        )

        encoder = import_path(
            "stage26_packet_encoder",
            (
                repo
                / "scripts/stage20_packet_image_encoder.py"
            ),
        )

        cnn_module = import_path(
            "stage26_cnn_module",
            (
                repo
                / "scripts/stage20_masked_cnn.py"
            ),
        )


    elif target == "STAGE21_MASKED_VIT_V1":

        import numpy as np
        import torch

        configure_torch_threads(
            thread_count
        )

        encoder = import_path(
            "stage26_packet_encoder",
            (
                repo
                / "scripts/stage20_packet_image_encoder.py"
            ),
        )

        vit_module = import_path(
            "stage26_vit_module",
            (
                repo
                / "scripts/stage21_masked_vit.py"
            ),
        )


    else:
        raise RuntimeError(
            f"Unknown target: {target}"
        )


    framework_import_end_ns = (
        perf_counter_ns()
    )


    # -------------------------------------------------------------------------
    # SYNTHETIC INPUT PREPARATION — supplementary cold component.
    # NOT part of primary model-deserialization or first-prediction duration.
    # -------------------------------------------------------------------------

    input_preparation_start_ns = (
        perf_counter_ns()
    )


    if target in {
        "STAGE16_XGBOOST_TUNED",
        "STAGE16_LIGHTGBM_TUNED",
        "STAGE16_CATBOOST_TUNED",
        "ENS_LGBM_XGB_EQUAL",
    }:

        Z, X_raw = build_group_a_input(
            repo=repo,
            seed=seed,
        )


    elif target in {
        "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
        "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
    }:

        Z = build_ft_input(
            seed=seed
        )


    elif target in {
        "STAGE20_MASKED_CNN_V1",
        "STAGE21_MASKED_VIT_V1",
    }:

        (
            image_uint8,
            image_scaled,
            padding_mask,
        ) = build_packet_input(
            encoder=encoder,
            seed=seed,
        )


    input_preparation_end_ns = (
        perf_counter_ns()
    )


    # -------------------------------------------------------------------------
    # MODEL DESERIALIZATION / CONSTRUCTION COMPONENT
    # -------------------------------------------------------------------------

    model_load_start_ns = (
        perf_counter_ns()
    )


    if target == "STAGE16_XGBOOST_TUNED":

        model = joblib.load(
            repo
            / "results/stage16_classical_benchmark_checkpoint/"
              "stage16_3_tuned_models/XGBOOST_tuned.joblib"
        )

        model.set_params(
            n_jobs=thread_count
        )


    elif target == "STAGE16_LIGHTGBM_TUNED":

        model = joblib.load(
            repo
            / "results/stage16_classical_benchmark_checkpoint/"
              "stage16_3_tuned_models/LIGHTGBM_tuned.joblib"
        )

        model.set_params(
            n_jobs=thread_count
        )


    elif target == "STAGE16_CATBOOST_TUNED":

        model = joblib.load(
            repo
            / "results/stage16_classical_benchmark_checkpoint/"
              "stage16_3_tuned_models/CATBOOST_tuned.joblib"
        )


    elif target == "ENS_LGBM_XGB_EQUAL":

        xgb_model = joblib.load(
            repo
            / "results/stage16_classical_benchmark_checkpoint/"
              "stage16_3_tuned_models/XGBOOST_tuned.joblib"
        )

        lgb_model = joblib.load(
            repo
            / "results/stage16_classical_benchmark_checkpoint/"
              "stage16_3_tuned_models/LIGHTGBM_tuned.joblib"
        )

        xgb_model.set_params(
            n_jobs=thread_count
        )

        lgb_model.set_params(
            n_jobs=thread_count
        )


    elif target in {
        "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
        "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
    }:

        arch_record = json.loads(
            (
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4c_frozen_architecture.json"
            ).read_text(
                encoding="utf-8"
            )
        )

        checkpoint_paths = {
            7:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4b_models/"
                  "FT_BALANCED_seed_7_best_extended.pt",

            29:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4a_models/"
                  "FT_BALANCED_seed_29_best.pt",

            101:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4a_models/"
                  "FT_BALANCED_seed_101_best.pt",

            313:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4c_models/"
                  "FT_BALANCED_seed_313_best.pt",

            997:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4c_models/"
                  "FT_BALANCED_seed_997_best.pt",
        }

        seeds = (
            [7, 29, 101, 313, 997]
            if target
            ==
            "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING"
            else
            [7]
        )

        ft_models = []

        for checkpoint_seed in seeds:

            ft_models.append(
                build_ft_model(
                    ft_module=ft_module,
                    arch_record=arch_record,
                    checkpoint=checkpoint_paths[
                        checkpoint_seed
                    ],
                )
            )


    elif target == "STAGE20_MASKED_CNN_V1":

        model = (
            cnn_module.Stage20MaskedCNNv1()
        )

        state = torch_load_state(
            repo
            / "results/stage20_1e_training/"
              "stage20_1e2_epoch10_model_state_dict.pt"
        )

        model.load_state_dict(
            state,
            strict=True,
        )

        model.eval()

        parameter_count = int(
            cnn_module.count_trainable_parameters(
                model
            )
        )

        if parameter_count != 93025:
            raise RuntimeError(
                f"CNN parameter mismatch: {parameter_count}"
            )


    elif target == "STAGE21_MASKED_VIT_V1":

        model = (
            vit_module.Stage21MaskedViTv1()
        )

        state = torch_load_state(
            repo
            / "results/stage21_architecture/"
              "stage21_2_epoch10_model_state_dict.pt"
        )

        model.load_state_dict(
            state,
            strict=True,
        )

        model.eval()

        parameter_count = int(
            vit_module.count_trainable_parameters(
                model
            )
        )

        if parameter_count != 91969:
            raise RuntimeError(
                f"ViT parameter mismatch: {parameter_count}"
            )


    model_load_end_ns = (
        perf_counter_ns()
    )


    # -------------------------------------------------------------------------
    # FIRST PREDICTION COMPONENT
    # Prepared model input -> materialized attack probability.
    # -------------------------------------------------------------------------

    first_prediction_start_ns = (
        perf_counter_ns()
    )


    if target == "STAGE16_XGBOOST_TUNED":

        probability = model.predict_proba(
            X_raw
        )[:, 1]


    elif target == "STAGE16_LIGHTGBM_TUNED":

        probability = model.predict_proba(
            X_raw
        )[:, 1]


    elif target == "STAGE16_CATBOOST_TUNED":

        probability = model.predict_proba(
            X_raw,
            thread_count=thread_count,
        )[:, 1]


    elif target == "ENS_LGBM_XGB_EQUAL":

        xgb_p = xgb_model.predict_proba(
            X_raw
        )[:, 1]

        lgb_p = lgb_model.predict_proba(
            X_raw
        )[:, 1]

        probability = (
            np.asarray(
                xgb_p,
                dtype=np.float64,
            )
            +
            np.asarray(
                lgb_p,
                dtype=np.float64,
            )
        ) / 2.0


    elif target in {
        "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
        "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
    }:

        x = torch.from_numpy(
            Z
        )

        member_probabilities = []

        with torch.inference_mode():

            for ft_model in ft_models:

                logits = ft_model(
                    x
                )

                member_probabilities.append(
                    torch.sigmoid(
                        logits
                    ).detach().cpu().numpy()
                )

        probability = np.mean(
            np.stack(
                member_probabilities,
                axis=0,
            ),
            axis=0,
        )


    elif target == "STAGE20_MASKED_CNN_V1":

        image = torch.from_numpy(
            image_uint8
        )

        mask = torch.from_numpy(
            padding_mask
        )

        with torch.inference_mode():

            logits = model(
                image,
                mask,
            )

            probability = torch.sigmoid(
                logits
            ).detach().cpu().numpy()


    elif target == "STAGE21_MASKED_VIT_V1":

        image = torch.from_numpy(
            image_scaled
        )

        mask = torch.from_numpy(
            padding_mask
        )

        with torch.inference_mode():

            logits = model(
                image,
                mask,
            )

            probability = torch.sigmoid(
                logits
            ).detach().cpu().numpy()


    # Materialization is complete here.
    first_prediction_end_ns = (
        perf_counter_ns()
    )


    p = validate_probability(
        probability
    )

    output_sha256 = sha256_array(
        p
    )


    result = {
        "schema":
            "stage26_1_cold_start_observation_v1",

        "status":
            "PASS",

        "target_id":
            target,

        "hardware_mode":
            "CPU_1_PHYSICAL_CORE",

        "thread_count":
            thread_count,

        "affinity_requested":
            affinity,

        "affinity_observed":
            observed_affinity,

        "batch_size":
            1,

        "output_sha256":
            output_sha256,

        "process_entry_ns":
            int(
                PROCESS_ENTRY_NS
            ),

        "worker_ready_ns":
            int(
                worker_ready_ns
            ),

        "framework_import_start_ns":
            int(
                framework_import_start_ns
            ),

        "framework_import_end_ns":
            int(
                framework_import_end_ns
            ),

        "input_preparation_start_ns":
            int(
                input_preparation_start_ns
            ),

        "input_preparation_end_ns":
            int(
                input_preparation_end_ns
            ),

        "model_load_start_ns":
            int(
                model_load_start_ns
            ),

        "model_load_end_ns":
            int(
                model_load_end_ns
            ),

        "first_prediction_start_ns":
            int(
                first_prediction_start_ns
            ),

        "first_prediction_end_ns":
            int(
                first_prediction_end_ns
            ),

        "process_spawn_to_worker_ready_ns":
            int(
                worker_ready_ns
                -
                parent_spawn_start_ns
            ),

        "framework_import_ns":
            int(
                framework_import_end_ns
                -
                framework_import_start_ns
            ),

        "input_preparation_ns":
            int(
                input_preparation_end_ns
                -
                input_preparation_start_ns
            ),

        "model_deserialization_load_ns":
            int(
                model_load_end_ns
                -
                model_load_start_ns
            ),

        "first_prediction_ns":
            int(
                first_prediction_end_ns
                -
                first_prediction_start_ns
            ),

        "spawn_to_first_output_ns":
            int(
                first_prediction_end_ns
                -
                parent_spawn_start_ns
            ),

        "holdout_accessed":
            False,

        "memory_profiled":
            False,

        "gpu_used":
            False,
    }


    for key in [
        "process_spawn_to_worker_ready_ns",
        "framework_import_ns",
        "input_preparation_ns",
        "model_deserialization_load_ns",
        "first_prediction_ns",
        "spawn_to_first_output_ns",
    ]:

        if result[
            key
        ] < 0:
            raise RuntimeError(
                f"Negative timing component: {key}"
            )


    print(
        json.dumps(
            result,
            sort_keys=True,
        ),
        flush=True,
    )


if __name__ == "__main__":
    main()
'''


WORKER_PATH.write_text(
    textwrap.dedent(
        worker_source
    ).lstrip(),
    encoding="utf-8",
)


worker_sha = sha256_file(
    WORKER_PATH
)


print(
    "Worker SHA256:",
    worker_sha,
)


# No memory profiling API is allowed in worker.
worker_text = WORKER_PATH.read_text(
    encoding="utf-8"
)


for forbidden in [
    "psutil",
    "ru_maxrss",
    "resource.getrusage",
    "memory_info(",
    "max_memory",
    "cuda.max_memory",
]:

    if forbidden in worker_text:
        raise RuntimeError(
            "Memory-profiling API found in cold-start worker: "
            + forbidden
        )


# =============================================================================
# 7. FREEZE COLD-START EXECUTION SCHEDULE BEFORE MEASUREMENT
# =============================================================================

banner(
    "STAGE26-1 :: FREEZE COLD-START EXECUTION SCHEDULE"
)


schedule_rows = []


for target in TARGETS:

    for repetition in range(
        1,
        COLD_REPETITIONS + 1,
    ):

        schedule_rows.append(
            {
                "target_id":
                    target,

                "repetition":
                    repetition,
            }
        )


rng = np.random.default_rng(
    MEASUREMENT_SEED
)

order = rng.permutation(
    len(
        schedule_rows
    )
)


frozen_schedule = []


for execution_order, original_index in enumerate(
    order,
    start=1,
):

    row = dict(
        schedule_rows[
            int(
                original_index
            )
        ]
    )

    row[
        "execution_order"
    ] = execution_order

    frozen_schedule.append(
        row
    )


schedule_payload = {
    "schema":
        "stage26_1_cold_start_schedule_v1",

    "parent_commit":
        EXPECTED_HEAD,

    "seed":
        MEASUREMENT_SEED,

    "target_count":
        len(
            TARGETS
        ),

    "repetitions_per_target":
        COLD_REPETITIONS,

    "total_repetitions":
        len(
            frozen_schedule
        ),

    "hardware_mode":
        "CPU_1_PHYSICAL_CORE",

    "affinity":
        PRIMARY_AFFINITY,

    "thread_count":
        PRIMARY_THREADS,

    "batch_size":
        BATCH_SIZE,

    "schedule":
        frozen_schedule,
}


write_json(
    SCHEDULE_PATH,
    schedule_payload,
)


schedule_sha = sha256_file(
    SCHEDULE_PATH
)


print(
    "Schedule SHA256:",
    schedule_sha,
)

print(
    "Total cold runs:",
    len(
        frozen_schedule
    ),
)

print(
    "\nFirst 12 frozen executions:"
)


for row in frozen_schedule[:12]:

    print(
        f"{row['execution_order']:03d}  "
        f"{row['target_id']:43s}  "
        f"rep={row['repetition']:02d}"
    )


# =============================================================================
# 8. FREEZE EXACT STAGE26-1 COMPONENT IMPLEMENTATION
# =============================================================================

banner(
    "STAGE26-1 :: FREEZE COLD COMPONENT IMPLEMENTATION"
)


implementation = {
    "schema":
        "stage26_1_cold_start_implementation_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-1",

    "parent_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA,

    "cpu_execution_plan_sha256":
        EXPECTED_PLAN_SHA,

    "preflight_receipt_sha256":
        EXPECTED_PREFLIGHT_RECEIPT_SHA,

    "worker_sha256":
        worker_sha,

    "schedule_sha256":
        schedule_sha,

    "hardware_mode":
        "CPU_1_PHYSICAL_CORE",

    "affinity":
        PRIMARY_AFFINITY,

    "thread_count":
        PRIMARY_THREADS,

    "batch_size":
        1,

    "repetitions_per_target":
        COLD_REPETITIONS,

    "execution_order":
        "DETERMINISTIC_RANDOMIZED_BEFORE_RESULTS",

    "clock":
        "time.perf_counter_ns",

    "component_boundaries": {
        "process_spawn_to_worker_ready_ns":
            (
                "Parent perf_counter_ns immediately before subprocess exec "
                "to child perf_counter_ns after config parse + CPU affinity "
                "and immediately before ML/framework imports."
            ),

        "framework_import_ns":
            (
                "Imports required numerical/model framework(s), frozen "
                "architecture module(s), thread setup, and packet encoder "
                "module where required."
            ),

        "input_preparation_ns":
            (
                "Supplementary diagnostic only: deterministic B=1 synthetic "
                "input construction. Not part of primary model-load or "
                "first-prediction component."
            ),

        "model_deserialization_load_ns":
            (
                "Frozen model construction/deserialization, state restoration, "
                "eval mode, and frozen inference thread configuration where "
                "model-specific."
            ),

        "first_prediction_ns":
            (
                "Prepared model input to materialized attack probability. "
                "Includes ensemble member forwards and probability averaging "
                "for frozen ensemble targets."
            ),

        "spawn_to_first_output_ns":
            (
                "Parent subprocess launch timestamp to materialized first "
                "attack-probability output in child."
            ),

        "parent_process_total_ns":
            (
                "Supplementary diagnostic: parent launch timestamp through "
                "worker process termination and stdout collection."
            ),
    },

    "environment_gate": {
        "cpu_utilization_percent_max":
            CPU_UTILIZATION_GATE_PERCENT,

        "cpu_utilization_sample_seconds":
            CPU_UTILIZATION_SAMPLE_SECONDS,

        "available_ram_gib_min":
            MIN_AVAILABLE_RAM_GIB,

        "retry_count":
            ENVIRONMENT_RETRY_COUNT,

        "retry_cooldown_seconds":
            ENVIRONMENT_RETRY_COOLDOWN_SECONDS,
    },

    "condition_cooldown_seconds":
        CONDITION_COOLDOWN_SECONDS,

    "subprocess_timeout_seconds":
        CONDITION_TIMEOUT_SECONDS,

    "raw_nanosecond_retention":
        True,

    "holdout_access":
        False,

    "memory_profiling":
        False,

    "gpu":
        False,

    "frozen_before_first_observation":
        True,
}


write_json(
    IMPLEMENTATION_PATH,
    implementation,
)


implementation_sha = sha256_file(
    IMPLEMENTATION_PATH
)


print(
    "Implementation SHA256:",
    implementation_sha,
)

print(
    "\n[PASS] Cold-start implementation fixed before first measurement."
)


# =============================================================================
# 9. ENVIRONMENT GATE
# =============================================================================

def environment_gate():

    attempts = []

    for attempt in range(
        1,
        ENVIRONMENT_RETRY_COUNT + 1,
    ):

        cpu_percent = float(
            psutil.cpu_percent(
                interval=CPU_UTILIZATION_SAMPLE_SECONDS
            )
        )

        vm = psutil.virtual_memory()

        available_ram_gib = float(
            vm.available
            /
            1024**3
        )

        passed = (
            cpu_percent
            <=
            CPU_UTILIZATION_GATE_PERCENT
            and
            available_ram_gib
            >=
            MIN_AVAILABLE_RAM_GIB
        )

        attempts.append(
            {
                "attempt":
                    attempt,

                "cpu_percent":
                    cpu_percent,

                "available_ram_gib":
                    available_ram_gib,

                "passed":
                    passed,
            }
        )

        if passed:
            return (
                True,
                attempts,
                attempts[-1],
            )

        if attempt < ENVIRONMENT_RETRY_COUNT:

            time.sleep(
                ENVIRONMENT_RETRY_COOLDOWN_SECONDS
            )

    return (
        False,
        attempts,
        attempts[-1],
    )


# =============================================================================
# 10. BEGIN FIRST STAGE26 PERFORMANCE MEASUREMENTS
# =============================================================================

banner(
    "STAGE26-1 :: BEGIN CPU COLD-START MEASUREMENTS"
)


print(
    "This is the FIRST Stage26 checkpoint permitted to observe performance."
)

print(
    "Fresh processes:",
    len(
        frozen_schedule
    ),
)

print(
    "Primary CPU condition: affinity=[0], threads=1, batch=1"
)


raw_rows = []


for schedule_index, schedule_row in enumerate(
    frozen_schedule,
):

    execution_order = schedule_row[
        "execution_order"
    ]

    target = schedule_row[
        "target_id"
    ]

    repetition = schedule_row[
        "repetition"
    ]


    # Frozen cooldown BETWEEN conditions.
    if schedule_index > 0:

        time.sleep(
            CONDITION_COOLDOWN_SECONDS
        )


    env_ok, env_attempts, env_final = (
        environment_gate()
    )


    if not env_ok:

        failure = {
            "schema":
                "stage26_1_environment_failure_v1",

            "status":
                "INVALID_ENVIRONMENT",

            "execution_order":
                execution_order,

            "target_id":
                target,

            "repetition":
                repetition,

            "environment_attempts":
                env_attempts,

            "performance_results_already_observed":
                len(
                    raw_rows
                ),
        }


        failure_path = (
            COLD_DIR
            / "stage26_1_environment_failure.json"
        )


        write_json(
            failure_path,
            failure,
        )


        raise RuntimeError(
            "INVALID_ENVIRONMENT after frozen retry rule. "
            "Do not rerun Stage26-1 silently. "
            f"Failure receipt: {failure_path}"
        )


    config = {
        "schema":
            "stage26_1_cold_worker_config_v1",

        "repo":
            str(
                REPO_DIR
            ),

        "target_id":
            target,

        "thread_count":
            PRIMARY_THREADS,

        "affinity":
            PRIMARY_AFFINITY,

        "measurement_seed":
            MEASUREMENT_SEED,
    }


    config_path = (
        COLD_DIR
        / "worker_config.json"
    )


    write_json(
        config_path,
        config,
    )


    env = os.environ.copy()


    for key in [
        "OMP_NUM_THREADS",
        "MKL_NUM_THREADS",
        "OPENBLAS_NUM_THREADS",
        "NUMEXPR_NUM_THREADS",
        "BLIS_NUM_THREADS",
        "VECLIB_MAXIMUM_THREADS",
    ]:

        env[
            key
        ] = "1"


    # Explicitly force CPU-only worker.
    env[
        "CUDA_VISIBLE_DEVICES"
    ] = ""


    parent_spawn_start_ns = (
        time.perf_counter_ns()
    )


    env[
        "STAGE26_PARENT_SPAWN_START_NS"
    ] = str(
        parent_spawn_start_ns
    )


    try:

        p = subprocess.run(
            [
                sys.executable,
                str(
                    WORKER_PATH
                ),
                str(
                    config_path
                ),
            ],
            cwd=REPO_DIR,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=False,
            timeout=CONDITION_TIMEOUT_SECONDS,
        )

    except subprocess.TimeoutExpired as exc:

        failure = {
            "schema":
                "stage26_1_timeout_failure_v1",

            "status":
                "TIMEOUT_RESOURCE_LIMIT",

            "execution_order":
                execution_order,

            "target_id":
                target,

            "repetition":
                repetition,

            "timeout_seconds":
                CONDITION_TIMEOUT_SECONDS,

            "performance_results_already_observed":
                len(
                    raw_rows
                ),
        }


        failure_path = (
            COLD_DIR
            / "stage26_1_timeout_failure.json"
        )


        write_json(
            failure_path,
            failure,
        )


        raise RuntimeError(
            "TIMEOUT_RESOURCE_LIMIT under frozen rule. "
            "Do not alter the timeout or iteration policy."
        ) from exc


    parent_process_end_ns = (
        time.perf_counter_ns()
    )


    if p.returncode != 0:

        failure = {
            "schema":
                "stage26_1_worker_failure_v1",

            "status":
                "WORKER_FAILURE",

            "execution_order":
                execution_order,

            "target_id":
                target,

            "repetition":
                repetition,

            "returncode":
                p.returncode,

            "stdout":
                p.stdout[-5000:],

            "stderr":
                p.stderr[-10000:],

            "performance_results_already_observed":
                len(
                    raw_rows
                ),
        }


        failure_path = (
            COLD_DIR
            / "stage26_1_worker_failure.json"
        )


        write_json(
            failure_path,
            failure,
        )


        print(
            "\nWorker stdout:"
        )

        print(
            p.stdout
        )

        print(
            "\nWorker stderr:"
        )

        print(
            p.stderr
        )


        raise RuntimeError(
            "Stage26-1 cold worker failed. "
            "Do not silently rerun."
        )


    lines = [
        line.strip()
        for line in p.stdout.splitlines()
        if line.strip()
    ]


    if not lines:
        raise RuntimeError(
            "Cold worker returned no JSON."
        )


    result = json.loads(
        lines[-1]
    )


    if result[
        "status"
    ] != "PASS":
        raise RuntimeError(
            "Cold worker did not return PASS."
        )


    # -------------------------------------------------------------------------
    # Prediction-integrity gate
    # -------------------------------------------------------------------------

    expected_prediction_sha = (
        expected_fingerprints[
            target
        ]
    )


    if (
        result[
            "output_sha256"
        ]
        !=
        expected_prediction_sha
    ):

        failure = {
            "schema":
                "stage26_1_prediction_integrity_failure_v1",

            "target_id":
                target,

            "repetition":
                repetition,

            "execution_order":
                execution_order,

            "expected_sha256":
                expected_prediction_sha,

            "actual_sha256":
                result[
                    "output_sha256"
                ],
        }


        failure_path = (
            COLD_DIR
            / "stage26_1_prediction_integrity_failure.json"
        )


        write_json(
            failure_path,
            failure,
        )


        raise RuntimeError(
            "Cold-start prediction fingerprint differs from frozen preflight."
        )


    parent_process_total_ns = int(
        parent_process_end_ns
        -
        parent_spawn_start_ns
    )


    row = {
        "execution_order":
            execution_order,

        "target_id":
            target,

        "repetition":
            repetition,

        "hardware_mode":
            "CPU_1_PHYSICAL_CORE",

        "thread_count":
            PRIMARY_THREADS,

        "affinity":
            "0",

        "batch_size":
            1,

        "environment_cpu_percent":
            env_final[
                "cpu_percent"
            ],

        "environment_available_ram_gib":
            env_final[
                "available_ram_gib"
            ],

        "environment_attempt_count":
            len(
                env_attempts
            ),

        "process_spawn_to_worker_ready_ns":
            result[
                "process_spawn_to_worker_ready_ns"
            ],

        "framework_import_ns":
            result[
                "framework_import_ns"
            ],

        "input_preparation_ns":
            result[
                "input_preparation_ns"
            ],

        "model_deserialization_load_ns":
            result[
                "model_deserialization_load_ns"
            ],

        "first_prediction_ns":
            result[
                "first_prediction_ns"
            ],

        "spawn_to_first_output_ns":
            result[
                "spawn_to_first_output_ns"
            ],

        "parent_process_total_ns":
            parent_process_total_ns,

        "output_sha256":
            result[
                "output_sha256"
            ],

        "status":
            "PASS",

        "holdout_accessed":
            False,

        "memory_profiled":
            False,

        "gpu_used":
            False,
    }


    raw_rows.append(
        row
    )


    # Durable raw observation immediately after every successful repetition.
    with RAW_JSONL.open(
        "a",
        encoding="utf-8",
    ) as f:

        f.write(
            json.dumps(
                row,
                sort_keys=True,
            )
            + "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )


    print(
        f"[PASS] {execution_order:03d}/"
        f"{len(frozen_schedule):03d}  "
        f"{target:43s} "
        f"rep={repetition:02d}  "
        f"cold_total="
        f"{row['spawn_to_first_output_ns'] / 1e6:9.3f} ms  "
        f"load="
        f"{row['model_deserialization_load_ns'] / 1e6:8.3f} ms  "
        f"first="
        f"{row['first_prediction_ns'] / 1e6:8.3f} ms"
    )


# =============================================================================
# 11. RAW COMPLETENESS GATE
# =============================================================================

banner(
    "STAGE26-1 :: RAW COMPLETENESS GATE"
)


if len(
    raw_rows
) != (
    len(
        TARGETS
    )
    *
    COLD_REPETITIONS
):

    raise RuntimeError(
        "Cold-start raw observation count mismatch."
    )


counts = {}


for row in raw_rows:

    counts[
        row[
            "target_id"
        ]
    ] = counts.get(
        row[
            "target_id"
        ],
        0,
    ) + 1


for target in TARGETS:

    print(
        f"{target:43s} "
        f"{counts.get(target, 0):2d}/20"
    )

    if counts.get(
        target,
        0,
    ) != 20:

        raise RuntimeError(
            f"Cold repetition count mismatch for {target}"
        )


# =============================================================================
# 12. SAVE RAW CSV
# =============================================================================

raw_columns = [
    "execution_order",
    "target_id",
    "repetition",
    "hardware_mode",
    "thread_count",
    "affinity",
    "batch_size",
    "environment_cpu_percent",
    "environment_available_ram_gib",
    "environment_attempt_count",
    "process_spawn_to_worker_ready_ns",
    "framework_import_ns",
    "input_preparation_ns",
    "model_deserialization_load_ns",
    "first_prediction_ns",
    "spawn_to_first_output_ns",
    "parent_process_total_ns",
    "output_sha256",
    "status",
    "holdout_accessed",
    "memory_profiled",
    "gpu_used",
]


with RAW_CSV.open(
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=raw_columns,
    )

    writer.writeheader()

    writer.writerows(
        raw_rows
    )


raw_csv_sha = sha256_file(
    RAW_CSV
)

raw_jsonl_sha = sha256_file(
    RAW_JSONL
)


# =============================================================================
# 13. COLD-START SUMMARIES + BOOTSTRAP CIs
# =============================================================================

banner(
    "STAGE26-1 :: COLD-START SUMMARY"
)


components = [
    "process_spawn_to_worker_ready_ns",
    "framework_import_ns",
    "input_preparation_ns",
    "model_deserialization_load_ns",
    "first_prediction_ns",
    "spawn_to_first_output_ns",
    "parent_process_total_ns",
]


summary_rows = []


for target in TARGETS:

    target_rows = [
        row
        for row in raw_rows
        if row[
            "target_id"
        ] == target
    ]


    for component in components:

        values_ns = np.asarray(
            [
                row[
                    component
                ]
                for row in target_rows
            ],
            dtype=np.float64,
        )


        values_ms = (
            values_ns
            /
            1_000_000.0
        )


        median_value = lambda x: float(
            np.percentile(
                x,
                50,
            )
        )

        p95_value = lambda x: float(
            np.percentile(
                x,
                95,
            )
        )


        median_ci = bootstrap_ci(
            values_ms,
            median_value,
            seed=stable_seed(
                target
                + "|"
                + component
                + "|p50"
            ),
        )

        p95_ci = bootstrap_ci(
            values_ms,
            p95_value,
            seed=stable_seed(
                target
                + "|"
                + component
                + "|p95"
            ),
        )


        summary_rows.append(
            {
                "target_id":
                    target,

                "component":
                    component,

                "n":
                    len(
                        values_ms
                    ),

                "mean_ms":
                    float(
                        np.mean(
                            values_ms
                        )
                    ),

                "std_ms":
                    float(
                        np.std(
                            values_ms,
                            ddof=1,
                        )
                    ),

                "p50_ms":
                    float(
                        np.percentile(
                            values_ms,
                            50,
                        )
                    ),

                "p95_ms":
                    float(
                        np.percentile(
                            values_ms,
                            95,
                        )
                    ),

                "min_ms":
                    float(
                        np.min(
                            values_ms
                        )
                    ),

                "max_ms":
                    float(
                        np.max(
                            values_ms
                        )
                    ),

                "p50_bootstrap_ci95_low_ms":
                    median_ci[0],

                "p50_bootstrap_ci95_high_ms":
                    median_ci[1],

                "p95_bootstrap_ci95_low_ms":
                    p95_ci[0],

                "p95_bootstrap_ci95_high_ms":
                    p95_ci[1],
            }
        )


summary_columns = list(
    summary_rows[0].keys()
)


with SUMMARY_CSV.open(
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=summary_columns,
    )

    writer.writeheader()

    writer.writerows(
        summary_rows
    )


summary_payload = {
    "schema":
        "stage26_1_cold_start_summary_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-1",

    "status":
        "COMPLETE",

    "parent_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA,

    "implementation_sha256":
        implementation_sha,

    "schedule_sha256":
        schedule_sha,

    "worker_sha256":
        worker_sha,

    "target_count":
        len(
            TARGETS
        ),

    "repetitions_per_target":
        COLD_REPETITIONS,

    "raw_observation_count":
        len(
            raw_rows
        ),

    "bootstrap_replicates":
        BOOTSTRAP_REPLICATES,

    "summary_rows":
        summary_rows,

    "claim_boundary":
        (
            "Cold-start resource profiling only. "
            "No warm-steady-state, throughput, memory, extraction, "
            "Pareto, or GPU conclusion is made at Stage26-1."
        ),
}


write_json(
    SUMMARY_JSON,
    summary_payload,
)


# Display only primary cold components.
primary_display_components = {
    "process_spawn_to_worker_ready_ns",
    "framework_import_ns",
    "model_deserialization_load_ns",
    "first_prediction_ns",
    "spawn_to_first_output_ns",
}


for target in TARGETS:

    print(
        "\n"
        + target
    )

    for row in summary_rows:

        if (
            row[
                "target_id"
            ] == target
            and
            row[
                "component"
            ] in primary_display_components
        ):

            label = (
                row[
                    "component"
                ]
                .replace(
                    "_ns",
                    "",
                )
            )

            print(
                f"  {label:36s} "
                f"p50={row['p50_ms']:10.3f} ms  "
                f"p95={row['p95_ms']:10.3f} ms  "
                f"mean={row['mean_ms']:10.3f} ms"
            )


# =============================================================================
# 14. BUILD RECEIPT
# =============================================================================

banner(
    "STAGE26-1 :: BUILD MEASUREMENT RECEIPT"
)


receipt = {
    "schema":
        "stage26_1_cpu_cold_start_receipt_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-1",

    "status":
        "PASS",

    "completed_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA,

    "cpu_execution_plan_sha256":
        EXPECTED_PLAN_SHA,

    "preflight_receipt_sha256":
        EXPECTED_PREFLIGHT_RECEIPT_SHA,

    "implementation_sha256":
        implementation_sha,

    "schedule_sha256":
        schedule_sha,

    "worker_sha256":
        worker_sha,

    "raw_jsonl_sha256":
        raw_jsonl_sha,

    "raw_csv_sha256":
        raw_csv_sha,

    "summary_csv_sha256":
        sha256_file(
            SUMMARY_CSV
        ),

    "summary_json_sha256":
        sha256_file(
            SUMMARY_JSON
        ),

    "hardware_mode":
        "CPU_1_PHYSICAL_CORE",

    "affinity":
        PRIMARY_AFFINITY,

    "thread_count":
        PRIMARY_THREADS,

    "batch_size":
        1,

    "target_count":
        len(
            TARGETS
        ),

    "repetitions_per_target":
        COLD_REPETITIONS,

    "raw_observation_count":
        len(
            raw_rows
        ),

    "performance_measurement_started":
        True,

    "cold_start_complete":
        True,

    "warm_latency_measured":
        False,

    "throughput_measured":
        False,

    "memory_profiled":
        False,

    "feature_extraction_measured":
        False,

    "holdout_reopened":
        False,

    "gpu_used":
        False,

    "scientific_boundary":
        (
            "Stage26-1 reports cold-start profiling only. "
            "No warm latency, throughput, memory, extraction, "
            "Pareto, bottleneck, capacity, or GPU claim is permitted yet."
        ),

    "next_action":
        (
            "GIT_ANCHOR_STAGE26_1_COLD_START_RESULTS_BEFORE_STAGE26_2"
        ),
}


write_json(
    RECEIPT_PATH,
    receipt,
)


receipt_sha = sha256_file(
    RECEIPT_PATH
)


print(
    "Receipt SHA256:",
    receipt_sha,
)


# =============================================================================
# 15. BUILD DURABLE REPOSITORY PACKAGE
# =============================================================================

banner(
    "STAGE26-1 :: BUILD DURABLE RESULTS PACKAGE"
)


REPO_COLD_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


files_to_copy = [
    WORKER_PATH,
    SCHEDULE_PATH,
    IMPLEMENTATION_PATH,
    RAW_JSONL,
    RAW_CSV,
    SUMMARY_CSV,
    SUMMARY_JSON,
    RECEIPT_PATH,
]


for source in files_to_copy:

    destination = (
        REPO_COLD_DIR
        / source.name
    )

    shutil.copy2(
        source,
        destination,
    )

    print(
        "[COPIED]",
        source.name,
    )


PACKAGE_MANIFEST = (
    REPO_COLD_DIR
    / "stage26_1_cold_start_package_manifest.json"
)


package_files = []


for path in sorted(
    REPO_COLD_DIR.iterdir()
):

    if (
        path.is_file()
        and
        path.name
        !=
        PACKAGE_MANIFEST.name
    ):

        package_files.append(
            {
                "path":
                    str(
                        path.relative_to(
                            REPO_DIR
                        )
                    ),

                "size_bytes":
                    path.stat().st_size,

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )


package_manifest = {
    "schema":
        "stage26_1_cold_start_package_manifest_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-1",

    "status":
        "READY_FOR_GIT_ANCHOR",

    "parent_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA,

    "receipt_sha256":
        receipt_sha,

    "file_count_excluding_manifest":
        len(
            package_files
        ),

    "files":
        package_files,

    "scientific_boundary": {
        "cold_start_measured":
            True,

        "warm_latency_measured":
            False,

        "throughput_measured":
            False,

        "memory_profiled":
            False,

        "feature_extraction_measured":
            False,

        "holdout_reopened":
            False,

        "gpu_used":
            False,
    },
}


write_json(
    PACKAGE_MANIFEST,
    package_manifest,
)


package_sha = sha256_file(
    PACKAGE_MANIFEST
)


print(
    "\nPackage manifest SHA256:",
    package_sha,
)


# =============================================================================
# 16. FINAL AUDIT
# =============================================================================

banner(
    "STAGE26-1 FINAL AUDIT"
)


status_after = git(
    "status",
    "--porcelain",
)


print(
    "Repository status:"
)

print(
    status_after
    if status_after
    else "<unexpected clean>"
)


if not status_after:
    raise RuntimeError(
        "Expected new Stage26-1 results package."
    )


unexpected = []


for line in status_after.splitlines():

    path = line[3:]

    if not path.startswith(
        "results/stage26_deployment_profiling/"
        "stage26_1_cpu_cold_start/"
    ):

        unexpected.append(
            line
        )


if unexpected:
    raise RuntimeError(
        "Unexpected repository changes outside Stage26-1:\n"
        + "\n".join(
            unexpected
        )
    )


# Confirm all prediction fingerprints remained frozen.
for target in TARGETS:

    target_hashes = {
        row[
            "output_sha256"
        ]
        for row in raw_rows
        if row[
            "target_id"
        ] == target
    }

    if target_hashes != {
        expected_fingerprints[
            target
        ]
    }:

        raise RuntimeError(
            f"Prediction-integrity final gate failed: {target}"
        )


banner(
    "STAGE26-1 CPU COLD-START PROFILING COMPLETE"
)


print(
    "Parent commit:"
)

print(
    " ",
    EXPECTED_HEAD,
)


print(
    "\nPerformance observations:"
)

print(
    " ",
    len(
        raw_rows
    ),
    "raw cold-start repetitions",
)


print(
    "\nTargets:"
)

print(
    " ",
    len(
        TARGETS
    ),
    "/ 8 complete",
)


print(
    "\nRepetitions:"
)

print(
    "  20 per target"
)


print(
    "\nCPU condition:"
)

print(
    "  CPU_1_PHYSICAL_CORE"
)

print(
    "  affinity = [0]"
)

print(
    "  threads  = 1"
)

print(
    "  batch    = 1"
)


print(
    "\nRAW JSONL SHA256:"
)

print(
    " ",
    raw_jsonl_sha,
)


print(
    "\nRAW CSV SHA256:"
)

print(
    " ",
    raw_csv_sha,
)


print(
    "\nRECEIPT SHA256:"
)

print(
    " ",
    receipt_sha,
)


print(
    "\nPACKAGE MANIFEST SHA256:"
)

print(
    " ",
    package_sha,
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  CPU COLD-START PROFILING COMPLETE"
)

print(
    "  RAW NANOSECOND OBSERVATIONS RETAINED"
)

print(
    "  PREDICTION INTEGRITY PRESERVED"
)

print(
    "  WARM LATENCY NOT YET MEASURED"
)

print(
    "  THROUGHPUT NOT YET MEASURED"
)

print(
    "  MEMORY NOT YET PROFILED"
)

print(
    "  FEATURE EXTRACTION NOT YET PROFILED"
)

print(
    "  HOLDOUT NOT REOPENED"
)

print(
    "  GPU NOT USED"
)


print(
    "\nCLAIM BOUNDARY:"
)

print(
    "  Do not infer deployment bottlenecks or Pareto dominance from"
)

print(
    "  Stage26-1 alone. These are cold-start measurements only."
)


print(
    "\nCRITICAL NEXT ACTION:"
)

print(
    "  STAGE26-1-GIT — commit, push, and remotely verify these raw"
)

print(
    "  cold-start measurements BEFORE Stage26-2 warm profiling."
)


STAGE26-1 :: PARENT / PROTOCOL GATE
Expected HEAD : c4502167547b7835ef4fa18ab9b1103b671e6894
Local HEAD    : c4502167547b7835ef4fa18ab9b1103b671e6894
origin/main   : c4502167547b7835ef4fa18ab9b1103b671e6894
Repository clean: True

Protocol SHA : d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
Plan SHA     : b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363
Preflight SHA: 4c7fc55a0e44e53587d385989dfec4f3f57f20768c7dd4a2d08e2936d2ba21e5

STAGE26-1 :: ANTI-DOUBLE-RUN GATE
[PASS] No previous Stage26-1 cold-start observations exist.

STAGE26-1 :: PREDICTION-INTEGRITY REFERENCE
STAGE16_XGBOOST_TUNED                       a0f5d59a72019a906426...
STAGE16_LIGHTGBM_TUNED                      fd7a0fbb059966d38d51...
STAGE16_CATBOOST_TUNED                      1316232b66e1cc125676...
FT_BALANCED_5_CHECKPOINT_SOFT_VOTING        aebbd0dc5e431cf6090d...
STAGE20_MASKED_CNN_V1                       0a4ea6d2334c1615b81d...
STAGE21_MASKED_VIT_V1                       567a

In [15]:
# =============================================================================
# STAGE26-1-GIT — ANCHOR CPU COLD-START RESULTS
#
# IDEMPOTENT:
#   - first run: commit + push + verify
#   - accidental second run: verify existing commit; do NOT create another
#
# THIS CELL:
#   - verifies the complete Stage26-1 package
#   - verifies all manifest-listed file hashes
#   - commits Stage26-1 if not already committed
#   - pushes only if origin/main is still the Stage26-0D parent
#   - verifies local commit == remote main
#   - verifies every Stage26-1 result file byte-for-byte remotely
#   - re-verifies original Stage26 protocol + preflight immutability
#
# THIS CELL DOES NOT:
#   - load models
#   - execute inference
#   - perform new timing
#   - perform memory profiling
#   - access holdout data
#   - use GPU
# =============================================================================

from __future__ import annotations

import os
import json
import stat
import hashlib
import tempfile
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient


# =============================================================================
# 0. FROZEN IDENTITIES
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT_HEAD = (
    "c4502167547b7835ef4fa18ab9b1103b671e6894"
)

COMMIT_MESSAGE = (
    "stage26: anchor CPU cold-start profiling"
)

STAGE26_1_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_1_cpu_cold_start"
)

MANIFEST_REL = (
    STAGE26_1_REL
    + "/stage26_1_cold_start_package_manifest.json"
)

RECEIPT_REL = (
    STAGE26_1_REL
    + "/stage26_1_cold_start_receipt.json"
)

RAW_JSONL_REL = (
    STAGE26_1_REL
    + "/stage26_1_cold_start_raw.jsonl"
)

RAW_CSV_REL = (
    STAGE26_1_REL
    + "/stage26_1_cold_start_raw.csv"
)


EXPECTED_MANIFEST_SHA256 = (
    "dd947e4964548058f48ecb5c2ec4b433bfeacf1c7bc477ced9afcc7874f4c678"
)

EXPECTED_RECEIPT_SHA256 = (
    "35d988fbdcf7d84d54e593cd37f1204a005c2e854ae7128cc559629616d99c10"
)

EXPECTED_RAW_JSONL_SHA256 = (
    "f10c275e1f9a4a91222328a4893f785d06048269dbed308ab20b45df6d4903dc"
)

EXPECTED_RAW_CSV_SHA256 = (
    "bd545c56dd197e076ee68d11cfc1d3cbadd3020044c5fe663f3eca6984059c3d"
)


PROTOCOL_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/measurement_protocol.json"
)

PLAN_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/cpu_execution_plan.json"
)

PREFLIGHT_RECEIPT_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0d_cpu_preflight/"
    "stage26_0d_cpu_preflight_receipt.json"
)


EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

EXPECTED_PLAN_SHA256 = (
    "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363"
)

EXPECTED_PREFLIGHT_SHA256 = (
    "4c7fc55a0e44e53587d385989dfec4f3f57f20768c7dd4a2d08e2936d2ba21e5"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):
    print("\n" + "=" * 106)
    print(text)
    print("=" * 106)


def run(
    cmd,
    *,
    env=None,
    check=True,
):
    p = subprocess.run(
        cmd,
        cwd=REPO_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "$ "
            + " ".join(cmd)
            + "\n\n"
            + p.stdout
        )

    return p


def git(
    *args,
    env=None,
    check=True,
):
    return run(
        ["git", *args],
        env=env,
        check=check,
    ).stdout.strip()


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def sha256_bytes(data):
    return hashlib.sha256(
        data
    ).hexdigest()


def git_blob_bytes(
    ref,
    path,
):
    p = subprocess.run(
        [
            "git",
            "show",
            f"{ref}:{path}",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stderr.decode(
                "utf-8",
                errors="replace",
            )
        )

    return p.stdout


# =============================================================================
# 2. LOCAL PACKAGE GATE
# =============================================================================

banner(
    "STAGE26-1-GIT :: LOCAL PACKAGE GATE"
)


manifest_path = (
    REPO_DIR
    / MANIFEST_REL
)

receipt_path = (
    REPO_DIR
    / RECEIPT_REL
)

raw_jsonl_path = (
    REPO_DIR
    / RAW_JSONL_REL
)

raw_csv_path = (
    REPO_DIR
    / RAW_CSV_REL
)


for path in [
    manifest_path,
    receipt_path,
    raw_jsonl_path,
    raw_csv_path,
]:

    if not path.exists():
        raise FileNotFoundError(
            path
        )


top_level_checks = [
    (
        "package manifest",
        manifest_path,
        EXPECTED_MANIFEST_SHA256,
    ),
    (
        "measurement receipt",
        receipt_path,
        EXPECTED_RECEIPT_SHA256,
    ),
    (
        "raw JSONL",
        raw_jsonl_path,
        EXPECTED_RAW_JSONL_SHA256,
    ),
    (
        "raw CSV",
        raw_csv_path,
        EXPECTED_RAW_CSV_SHA256,
    ),
]


for name, path, expected_sha in top_level_checks:

    actual_sha = sha256_file(
        path
    )

    ok = (
        actual_sha
        ==
        expected_sha
    )

    print(
        f"{name:24s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual_sha}"
    )

    if not ok:
        raise RuntimeError(
            f"Stage26-1 {name} SHA mismatch."
        )


# =============================================================================
# 3. MANIFEST-COMPLETE LOCAL HASH VERIFICATION
# =============================================================================

banner(
    "STAGE26-1-GIT :: MANIFEST-COMPLETE LOCAL HASH GATE"
)


manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)


if manifest.get(
    "status"
) != "READY_FOR_GIT_ANCHOR":

    raise RuntimeError(
        "Stage26-1 manifest is not READY_FOR_GIT_ANCHOR."
    )


if manifest.get(
    "parent_commit"
) != EXPECTED_PARENT_HEAD:

    raise RuntimeError(
        "Stage26-1 manifest parent mismatch."
    )


manifest_files = manifest[
    "files"
]


print(
    "Manifest-listed result files:",
    len(
        manifest_files
    ),
)


for record in manifest_files:

    relpath = record[
        "path"
    ]

    path = (
        REPO_DIR
        / relpath
    )

    expected_sha = record[
        "sha256"
    ]

    expected_size = int(
        record[
            "size_bytes"
        ]
    )


    if not path.exists():
        raise FileNotFoundError(
            path
        )


    actual_sha = sha256_file(
        path
    )

    actual_size = (
        path.stat().st_size
    )


    sha_ok = (
        actual_sha
        ==
        expected_sha
    )

    size_ok = (
        actual_size
        ==
        expected_size
    )


    print(
        f"{Path(relpath).name:48s} "
        f"sha={'PASS' if sha_ok else 'FAIL'} "
        f"size={'PASS' if size_ok else 'FAIL'}"
    )


    if not sha_ok or not size_ok:

        raise RuntimeError(
            f"Manifest verification failed: {relpath}"
        )


# =============================================================================
# 4. RECEIPT CONTENT GATE
# =============================================================================

banner(
    "STAGE26-1-GIT :: SCIENTIFIC RECEIPT GATE"
)


receipt = json.loads(
    receipt_path.read_text(
        encoding="utf-8"
    )
)


checks = {
    "status":
        "PASS",

    "target_count":
        8,

    "repetitions_per_target":
        20,

    "raw_observation_count":
        160,

    "performance_measurement_started":
        True,

    "cold_start_complete":
        True,

    "warm_latency_measured":
        False,

    "throughput_measured":
        False,

    "memory_profiled":
        False,

    "feature_extraction_measured":
        False,

    "holdout_reopened":
        False,

    "gpu_used":
        False,
}


for key, expected in checks.items():

    actual = receipt.get(
        key
    )

    print(
        f"{key:36s}: "
        f"{actual!r}"
    )

    if actual != expected:
        raise RuntimeError(
            f"Receipt scientific-boundary mismatch: "
            f"{key}={actual!r}; expected {expected!r}"
        )


if receipt.get(
    "parent_commit"
) != EXPECTED_PARENT_HEAD:

    raise RuntimeError(
        "Cold-start receipt parent mismatch."
    )


print(
    "\n[PASS] Stage26-1 contains exactly 160 cold-start observations."
)

print(
    "[PASS] Warm latency / throughput / memory remain unmeasured."
)

print(
    "[PASS] Holdout remained closed and GPU remained unused."
)


# =============================================================================
# 5. CURRENT GIT STATE — IDEMPOTENT COMMIT LOGIC
# =============================================================================

banner(
    "STAGE26-1-GIT :: CURRENT GIT STATE"
)


HEAD_BEFORE = git(
    "rev-parse",
    "HEAD",
)

STATUS_BEFORE = git(
    "status",
    "--porcelain",
)


print(
    "Current HEAD:",
    HEAD_BEFORE,
)

print(
    "Repository clean:",
    STATUS_BEFORE == "",
)


if STATUS_BEFORE:

    print(
        "\nRepository status:"
    )

    print(
        STATUS_BEFORE
    )


stage26_1_already_committed = False


if HEAD_BEFORE != EXPECTED_PARENT_HEAD:

    current_parent = git(
        "rev-parse",
        "HEAD^",
        check=False,
    )

    current_subject = git(
        "log",
        "-1",
        "--pretty=%s",
    )


    if (
        current_parent
        ==
        EXPECTED_PARENT_HEAD
        and
        current_subject
        ==
        COMMIT_MESSAGE
    ):

        stage26_1_already_committed = True

        print(
            "\n[INFO] Stage26-1 commit already exists locally."
        )


    else:

        raise RuntimeError(
            "Current HEAD is neither the expected Stage26-0D parent "
            "nor an idempotent Stage26-1 result commit."
        )


# =============================================================================
# 6. COMMIT IF NEEDED
# =============================================================================

if not stage26_1_already_committed:

    banner(
        "STAGE26-1-GIT :: STAGE + COMMIT"
    )


    if not STATUS_BEFORE:

        raise RuntimeError(
            "Expected uncommitted Stage26-1 package but worktree is clean."
        )


    unexpected = []


    for line in STATUS_BEFORE.splitlines():

        path = line[3:]


        if not path.startswith(
            STAGE26_1_REL
            + "/"
        ):

            unexpected.append(
                line
            )


    if unexpected:

        raise RuntimeError(
            "Unexpected repository modifications outside Stage26-1:\n"
            +
            "\n".join(
                unexpected
            )
        )


    git(
        "add",
        "--",
        STAGE26_1_REL,
    )


    staged = git(
        "diff",
        "--cached",
        "--name-status",
    )


    print(
        staged
    )


    if not staged:

        raise RuntimeError(
            "Nothing staged for Stage26-1."
        )


    for line in staged.splitlines():

        relpath = line.split(
            "\t"
        )[-1]


        if not relpath.startswith(
            STAGE26_1_REL
            + "/"
        ):

            raise RuntimeError(
                "Unexpected staged file:\n"
                + line
            )


    # Git identity only if missing.
    if not git(
        "config",
        "--get",
        "user.name",
        check=False,
    ):

        git(
            "config",
            "user.name",
            "themubasshir",
        )


    if not git(
        "config",
        "--get",
        "user.email",
        check=False,
    ):

        git(
            "config",
            "user.email",
            "themubasshir@users.noreply.github.com",
        )


    commit_output = git(
        "commit",
        "-m",
        COMMIT_MESSAGE,
    )


    print(
        "\n"
        + commit_output
    )


    STAGE26_1_HEAD = git(
        "rev-parse",
        "HEAD",
    )


    parent = git(
        "rev-parse",
        "HEAD^",
    )


    if parent != EXPECTED_PARENT_HEAD:

        raise RuntimeError(
            "Stage26-1 commit has incorrect parent."
        )


else:

    STAGE26_1_HEAD = HEAD_BEFORE


print(
    "\nStage26-1 commit:",
    STAGE26_1_HEAD,
)


# =============================================================================
# 7. VERIFY COMMITTED FILE BYTES LOCALLY
# =============================================================================

banner(
    "STAGE26-1-GIT :: COMMITTED-BLOB HASH GATE"
)


# Manifest itself.
committed_manifest_sha = sha256_bytes(
    git_blob_bytes(
        "HEAD",
        MANIFEST_REL,
    )
)


print(
    f"{Path(MANIFEST_REL).name:48s} "
    f"{'PASS' if committed_manifest_sha == EXPECTED_MANIFEST_SHA256 else 'FAIL'} "
    f"{committed_manifest_sha}"
)


if (
    committed_manifest_sha
    !=
    EXPECTED_MANIFEST_SHA256
):

    raise RuntimeError(
        "Committed Stage26-1 manifest mismatch."
    )


# Every file listed inside package manifest.
for record in manifest_files:

    relpath = record[
        "path"
    ]

    expected_sha = record[
        "sha256"
    ]


    committed_sha = sha256_bytes(
        git_blob_bytes(
            "HEAD",
            relpath,
        )
    )


    ok = (
        committed_sha
        ==
        expected_sha
    )


    print(
        f"{Path(relpath).name:48s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{committed_sha}"
    )


    if not ok:

        raise RuntimeError(
            f"Committed blob mismatch: {relpath}"
        )


# =============================================================================
# 8. FETCH CURRENT REMOTE
# =============================================================================

banner(
    "STAGE26-1-GIT :: REMOTE RELATIONSHIP"
)


git(
    "fetch",
    "origin",
    "main",
)


REMOTE_BEFORE = git(
    "rev-parse",
    "origin/main",
)


print(
    "Stage26-1 local HEAD:",
    STAGE26_1_HEAD,
)

print(
    "origin/main currently:",
    REMOTE_BEFORE,
)


if REMOTE_BEFORE == STAGE26_1_HEAD:

    relationship = (
        "REMOTE_ALREADY_HAS_STAGE26_1"
    )


elif REMOTE_BEFORE == EXPECTED_PARENT_HEAD:

    relationship = (
        "LOCAL_STAGE26_1_AHEAD_BY_ONE"
    )


else:

    relationship = (
        "UNEXPECTED_REMOTE_DIVERGENCE"
    )


print(
    "Relationship:",
    relationship,
)


if (
    relationship
    ==
    "UNEXPECTED_REMOTE_DIVERGENCE"
):

    raise RuntimeError(
        "origin/main diverged unexpectedly. "
        "Do NOT automatically push."
    )


# =============================================================================
# 9. PUSH ONLY IF REQUIRED
# =============================================================================

if (
    relationship
    ==
    "LOCAL_STAGE26_1_AHEAD_BY_ONE"
):

    banner(
        "STAGE26-1-GIT :: PUSH"
    )


    secret_client = (
        UserSecretsClient()
    )


    aliases = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "gh_token",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
        "gh_pat",
    ]


    token = None
    token_label = None


    for label in aliases:

        try:

            value = (
                secret_client.get_secret(
                    label
                )
            )

        except Exception:

            value = None


        if value:

            token = value.strip()

            token_label = label

            break


    if not token:

        raise RuntimeError(
            "No usable GitHub credential in Kaggle Secrets."
        )


    print(
        f"[FOUND] {token_label} "
        f"({len(token)} characters)"
    )

    print(
        "Token value will not be printed."
    )


    fd, askpass_name = (
        tempfile.mkstemp(
            prefix="stage26_1_askpass_",
            suffix=".sh",
        )
    )

    os.close(
        fd
    )


    askpass = Path(
        askpass_name
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' "x-access-token" ;;
  *Password*) printf '%s\\n' "$GITHUB_TOKEN" ;;
  *)          printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        stat.S_IRUSR
        |
        stat.S_IWUSR
        |
        stat.S_IXUSR
    )


    push_env = (
        os.environ.copy()
    )


    push_env[
        "GITHUB_TOKEN"
    ] = token

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"


    try:

        push_result = run(
            [
                "git",
                "push",
                "origin",
                "HEAD:main",
            ],
            env=push_env,
        )


        print(
            push_result.stdout
        )


    finally:

        try:
            askpass.unlink()

        except FileNotFoundError:
            pass


        token = None


# =============================================================================
# 10. FETCH BACK + REMOTE COMMIT GATE
# =============================================================================

banner(
    "STAGE26-1-GIT :: REMOTE COMMIT GATE"
)


git(
    "fetch",
    "origin",
    "main",
)


LOCAL_HEAD = git(
    "rev-parse",
    "HEAD",
)

REMOTE_HEAD = git(
    "rev-parse",
    "origin/main",
)


print(
    "Local HEAD :",
    LOCAL_HEAD,
)

print(
    "Remote HEAD:",
    REMOTE_HEAD,
)


if LOCAL_HEAD != STAGE26_1_HEAD:

    raise RuntimeError(
        "Local HEAD changed unexpectedly."
    )


if REMOTE_HEAD != STAGE26_1_HEAD:

    raise RuntimeError(
        "origin/main does not match Stage26-1 commit."
    )


ls_remote = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


ls_remote_sha = (
    ls_remote.split()[0]
    if ls_remote
    else None
)


print(
    "ls-remote :",
    ls_remote_sha,
)


if ls_remote_sha != STAGE26_1_HEAD:

    raise RuntimeError(
        "Independent remote ref verification failed."
    )


# =============================================================================
# 11. REMOTE BYTE-FOR-BYTE PACKAGE VERIFICATION
# =============================================================================

banner(
    "STAGE26-1-GIT :: REMOTE PACKAGE SHA GATE"
)


remote_manifest_sha = sha256_bytes(
    git_blob_bytes(
        "origin/main",
        MANIFEST_REL,
    )
)


manifest_ok = (
    remote_manifest_sha
    ==
    EXPECTED_MANIFEST_SHA256
)


print(
    f"{Path(MANIFEST_REL).name:48s} "
    f"{'PASS' if manifest_ok else 'FAIL'} "
    f"{remote_manifest_sha}"
)


if not manifest_ok:

    raise RuntimeError(
        "Remote Stage26-1 package manifest mismatch."
    )


for record in manifest_files:

    relpath = record[
        "path"
    ]

    expected_sha = record[
        "sha256"
    ]


    remote_sha = sha256_bytes(
        git_blob_bytes(
            "origin/main",
            relpath,
        )
    )


    ok = (
        remote_sha
        ==
        expected_sha
    )


    print(
        f"{Path(relpath).name:48s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{remote_sha}"
    )


    if not ok:

        raise RuntimeError(
            f"Remote Stage26-1 file mismatch: {relpath}"
        )


# =============================================================================
# 12. EXPLICIT PRIMARY RAW-RESULT REMOTE HASH GATES
# =============================================================================

banner(
    "STAGE26-1-GIT :: PRIMARY RESULT HASH GATE"
)


explicit_remote_checks = [
    (
        "receipt",
        RECEIPT_REL,
        EXPECTED_RECEIPT_SHA256,
    ),
    (
        "raw_jsonl",
        RAW_JSONL_REL,
        EXPECTED_RAW_JSONL_SHA256,
    ),
    (
        "raw_csv",
        RAW_CSV_REL,
        EXPECTED_RAW_CSV_SHA256,
    ),
]


for name, relpath, expected_sha in explicit_remote_checks:

    remote_sha = sha256_bytes(
        git_blob_bytes(
            "origin/main",
            relpath,
        )
    )


    ok = (
        remote_sha
        ==
        expected_sha
    )


    print(
        f"{name:14s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{remote_sha}"
    )


    if not ok:

        raise RuntimeError(
            f"Remote primary Stage26-1 result mismatch: {name}"
        )


# =============================================================================
# 13. REVERIFY ORIGINAL SCIENTIFIC LOCKS
# =============================================================================

banner(
    "STAGE26-1-GIT :: UPSTREAM IMMUTABILITY GATE"
)


upstream_checks = [
    (
        "measurement_protocol",
        PROTOCOL_REL,
        EXPECTED_PROTOCOL_SHA256,
    ),
    (
        "cpu_execution_plan",
        PLAN_REL,
        EXPECTED_PLAN_SHA256,
    ),
    (
        "cpu_preflight_receipt",
        PREFLIGHT_RECEIPT_REL,
        EXPECTED_PREFLIGHT_SHA256,
    ),
]


for name, relpath, expected_sha in upstream_checks:

    remote_sha = sha256_bytes(
        git_blob_bytes(
            "origin/main",
            relpath,
        )
    )


    ok = (
        remote_sha
        ==
        expected_sha
    )


    print(
        f"{name:28s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{remote_sha}"
    )


    if not ok:

        raise RuntimeError(
            f"Upstream Stage26 lock changed: {name}"
        )


# =============================================================================
# 14. FINAL CLEANLINESS
# =============================================================================

banner(
    "STAGE26-1-GIT :: FINAL AUDIT"
)


FINAL_STATUS = git(
    "status",
    "--porcelain",
)


print(
    "Repository clean:",
    FINAL_STATUS == "",
)


if FINAL_STATUS:

    print(
        FINAL_STATUS
    )

    raise RuntimeError(
        "Repository is dirty after Stage26-1 anchor."
    )


# Confirm commit ancestry.
FINAL_PARENT = git(
    "rev-parse",
    "HEAD^",
)


if FINAL_PARENT != EXPECTED_PARENT_HEAD:

    raise RuntimeError(
        "Stage26-1 result commit has wrong parent."
    )


FINAL_SUBJECT = git(
    "log",
    "-1",
    "--pretty=%s",
)


if FINAL_SUBJECT != COMMIT_MESSAGE:

    raise RuntimeError(
        "Stage26-1 commit subject mismatch."
    )


# =============================================================================
# 15. CLOSURE
# =============================================================================

banner(
    "STAGE26-1-GIT COMPLETE"
)


print(
    "STAGE26-0D PARENT:"
)

print(
    " ",
    EXPECTED_PARENT_HEAD,
)


print(
    "\nSTAGE26-1 COLD-START COMMIT:"
)

print(
    " ",
    STAGE26_1_HEAD,
)


print(
    "\norigin/main:"
)

print(
    " ",
    REMOTE_HEAD,
)


print(
    "\nRAW COLD-START OBSERVATIONS:"
)

print(
    "  160 / 160"
)


print(
    "\nRAW JSONL SHA256:"
)

print(
    " ",
    EXPECTED_RAW_JSONL_SHA256,
)


print(
    "\nRAW CSV SHA256:"
)

print(
    " ",
    EXPECTED_RAW_CSV_SHA256,
)


print(
    "\nRECEIPT SHA256:"
)

print(
    " ",
    EXPECTED_RECEIPT_SHA256,
)


print(
    "\nPACKAGE MANIFEST SHA256:"
)

print(
    " ",
    EXPECTED_MANIFEST_SHA256,
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  COMMIT MATCH                    : PASS"
)

print(
    "  LS-REMOTE                       : PASS"
)

print(
    "  ALL MANIFEST FILES              : PASS"
)

print(
    "  RAW JSONL                       : PASS"
)

print(
    "  RAW CSV                         : PASS"
)

print(
    "  COLD-START RECEIPT              : PASS"
)

print(
    "  ORIGINAL MEASUREMENT PROTOCOL   : PASS"
)

print(
    "  ORIGINAL CPU EXECUTION PLAN     : PASS"
)

print(
    "  ORIGINAL CPU PREFLIGHT RECEIPT  : PASS"
)

print(
    "  REPOSITORY CLEAN                : PASS"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  CPU COLD-START RESULTS DURABLY FROZEN"
)

print(
    "  160 RAW COLD-START OBSERVATIONS PRESERVED"
)

print(
    "  WARM LATENCY NOT YET MEASURED"
)

print(
    "  THROUGHPUT NOT YET MEASURED"
)

print(
    "  MEMORY NOT YET PROFILED"
)

print(
    "  FEATURE EXTRACTION NOT YET PROFILED"
)

print(
    "  HOLDOUT NOT REOPENED"
)

print(
    "  GPU NOT USED"
)


print(
    "\nNEXT:"
)

print(
    "  STAGE26-2 — WARM CPU INFERENCE LATENCY + BATCH THROUGHPUT"
)

print(
    "  Execute the already-frozen 80-condition randomized CPU plan."
)

print(
    "  Raw per-iteration nanosecond timings will be retained."
)


STAGE26-1-GIT :: LOCAL PACKAGE GATE
package manifest         PASS dd947e4964548058f48ecb5c2ec4b433bfeacf1c7bc477ced9afcc7874f4c678
measurement receipt      PASS 35d988fbdcf7d84d54e593cd37f1204a005c2e854ae7128cc559629616d99c10
raw JSONL                PASS f10c275e1f9a4a91222328a4893f785d06048269dbed308ab20b45df6d4903dc
raw CSV                  PASS bd545c56dd197e076ee68d11cfc1d3cbadd3020044c5fe663f3eca6984059c3d

STAGE26-1-GIT :: MANIFEST-COMPLETE LOCAL HASH GATE
Manifest-listed result files: 8
stage26_1_cold_start_implementation.json         sha=PASS size=PASS
stage26_1_cold_start_raw.csv                     sha=PASS size=PASS
stage26_1_cold_start_raw.jsonl                   sha=PASS size=PASS
stage26_1_cold_start_receipt.json                sha=PASS size=PASS
stage26_1_cold_start_schedule.json               sha=PASS size=PASS
stage26_1_cold_start_summary.csv                 sha=PASS size=PASS
stage26_1_cold_start_summary.json                sha=PASS size=PASS
stage26_cold_start_work

In [16]:
# =============================================================================
# STAGE26-2 — WARM CPU INFERENCE LATENCY + BATCH THROUGHPUT
#
# PARENT COMMIT:
#   46379b6d036008db4d60b056a66f4c01383e3298
#
# EXECUTES THE ALREADY-FROZEN 80-CONDITION CPU PLAN:
#
#   8 targets
#   × 2 CPU execution modes
#   × 5 batch sizes
#   = 80 conditions
#
# BATCH / REPEAT POLICY — ALREADY FROZEN
# ---------------------------------------
# B=1     warmup=50  timed=200   LATENCY_PRIMARY
# B=64    warmup=30  timed=150   MICROBATCH
# B=256   warmup=20  timed=100   MICROBATCH
# B=1024  warmup=10  timed=50    THROUGHPUT
# B=8192  warmup=5   timed=20    THROUGHPUT_STRESS
#
# PRIMARY MEASUREMENT:
#   prepared model input -> materialized attack probability
#
# RAW MEASUREMENT:
#   one elapsed_ns observation PER timed inference iteration
#
# FAILURE POLICY — ALREADY FROZEN
# --------------------------------
# OOM:
#   RESOURCE_LIMIT_OOM
#   condition retained; batch is NOT reduced
#
# timeout:
#   TIMEOUT_RESOURCE_LIMIT
#   condition retained; timeout/iterations are NOT changed
#
# environment failure:
#   INVALID_ENVIRONMENT
#   abort checkpoint; do not silently benchmark
#
# implementation/backend/software failure:
#   WORKER_FAILURE
#   abort checkpoint for scoped diagnosis
#
# MEMORY:
#   NOT measured here.
#
# GPU:
#   OFF.
#
# HOLDOUT:
#   NOT accessed.
#
# RESUME SAFETY:
#   Completed condition receipts are never remeasured.
#   If the notebook/runtime is interrupted, exact unfinished conditions can
#   be resumed without replacing already-observed measurements.
# =============================================================================

from __future__ import annotations

import os
import sys
import csv
import json
import time
import signal
import shutil
import hashlib
import subprocess
import textwrap
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import psutil


# =============================================================================
# 0. FROZEN CONSTANTS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

WORKERS_DIR = (
    STAGE26_ROOT
    / "workers"
)

STEADY_DIR = (
    STAGE26_ROOT
    / "steady_state"
)

CONDITION_DIR = (
    STEADY_DIR
    / "condition_receipts"
)

CONFIG_DIR = (
    STEADY_DIR
    / "condition_configs"
)

REPO_STAGE26_2_DIR = (
    REPO_DIR
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_2_cpu_warm_inference"
)


for d in [
    WORKERS_DIR,
    STEADY_DIR,
    CONDITION_DIR,
    CONFIG_DIR,
]:
    d.mkdir(
        parents=True,
        exist_ok=True,
    )


EXPECTED_HEAD = (
    "46379b6d036008db4d60b056a66f4c01383e3298"
)

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

EXPECTED_EXECUTION_PLAN_SHA256 = (
    "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363"
)

EXPECTED_PREFLIGHT_RECEIPT_SHA256 = (
    "4c7fc55a0e44e53587d385989dfec4f3f57f20768c7dd4a2d08e2936d2ba21e5"
)

EXPECTED_COLD_RECEIPT_SHA256 = (
    "35d988fbdcf7d84d54e593cd37f1204a005c2e854ae7128cc559629616d99c10"
)

MEASUREMENT_SEED = 26042

BOOTSTRAP_SEED = 26042

BOOTSTRAP_REPLICATES = 2000

CPU_UTILIZATION_GATE_PERCENT = 20.0

MIN_AVAILABLE_RAM_GIB = 8.0

ENVIRONMENT_RETRY_COUNT = 3

ENVIRONMENT_RETRY_COOLDOWN_SECONDS = 5

CPU_UTILIZATION_SAMPLE_SECONDS = 1.0

CONDITION_COOLDOWN_SECONDS = 2

CONDITION_TIMEOUT_SECONDS = 600


ALLOWED_FINAL_CONDITION_STATUSES = {
    "PASS",
    "RESOURCE_LIMIT_OOM",
    "TIMEOUT_RESOURCE_LIMIT",
}


# =============================================================================
# 1. FROZEN FILE PATHS
# =============================================================================

PROTOCOL_PATH = (
    REPO_DIR
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXECUTION_PLAN_PATH = (
    REPO_DIR
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "cpu_execution_plan.json"
)

PREFLIGHT_RECEIPT_PATH = (
    REPO_DIR
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0d_cpu_preflight"
    / "stage26_0d_cpu_preflight_receipt.json"
)

COLD_RECEIPT_PATH = (
    REPO_DIR
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_1_cpu_cold_start"
    / "stage26_1_cold_start_receipt.json"
)


WORKER_PATH = (
    WORKERS_DIR
    / "stage26_warm_cpu_worker.py"
)

IMPLEMENTATION_PATH = (
    STEADY_DIR
    / "stage26_2_warm_cpu_implementation.json"
)

CONDITION_RECEIPTS_JSON = (
    STEADY_DIR
    / "stage26_2_condition_receipts.json"
)

CONDITION_STATUS_CSV = (
    STEADY_DIR
    / "stage26_2_condition_status.csv"
)

RAW_JSONL = (
    STEADY_DIR
    / "stage26_2_warm_raw.jsonl"
)

RAW_CSV = (
    STEADY_DIR
    / "stage26_2_warm_raw.csv"
)

SUMMARY_CSV = (
    STEADY_DIR
    / "stage26_2_warm_summary.csv"
)

SUMMARY_JSON = (
    STEADY_DIR
    / "stage26_2_warm_summary.json"
)

RECEIPT_PATH = (
    STEADY_DIR
    / "stage26_2_warm_cpu_receipt.json"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def banner(text: str):
    print(
        "\n"
        + "=" * 112
    )

    print(text)

    print(
        "=" * 112
    )


def git(*args):
    p = subprocess.run(
        [
            "git",
            *args,
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(path: Path):
    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            b = f.read(
                8 * 1024 * 1024
            )

            if not b:
                break

            h.update(b)

    return h.hexdigest()


def sha256_bytes(data: bytes):
    return hashlib.sha256(
        data
    ).hexdigest()


def write_json(
    path: Path,
    obj,
):

    tmp = Path(
        str(path)
        + ".tmp"
    )

    raw = (
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
        )
        + "\n"
    ).encode(
        "utf-8"
    )

    with tmp.open(
        "wb"
    ) as f:

        f.write(raw)

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def stable_seed(text: str):
    digest = hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).digest()

    return int.from_bytes(
        digest[:8],
        "little",
    ) % (
        2**32
    )


def bootstrap_ci(
    values,
    statistic,
    *,
    seed=BOOTSTRAP_SEED,
    replicates=BOOTSTRAP_REPLICATES,
):

    values = np.asarray(
        values,
        dtype=np.float64,
    )

    if values.size == 0:
        return (
            None,
            None,
        )

    rng = np.random.default_rng(
        int(seed)
    )

    n = int(
        values.size
    )

    estimates = np.empty(
        int(
            replicates
        ),
        dtype=np.float64,
    )

    for i in range(
        int(
            replicates
        )
    ):

        sample = values[
            rng.integers(
                0,
                n,
                size=n,
            )
        ]

        estimates[i] = statistic(
            sample
        )

    return (
        float(
            np.percentile(
                estimates,
                2.5,
            )
        ),
        float(
            np.percentile(
                estimates,
                97.5,
            )
        ),
    )


# =============================================================================
# 3. PARENT / PROTOCOL / REPOSITORY GATE
# =============================================================================

banner(
    "STAGE26-2 :: PARENT / PROTOCOL GATE"
)


HEAD = git(
    "rev-parse",
    "HEAD",
)

REMOTE = git(
    "rev-parse",
    "origin/main",
)

STATUS = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD :",
    EXPECTED_HEAD,
)

print(
    "Local HEAD    :",
    HEAD,
)

print(
    "origin/main   :",
    REMOTE,
)

print(
    "Repository clean:",
    STATUS == "",
)


if HEAD != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected local HEAD before Stage26-2."
    )


if REMOTE != EXPECTED_HEAD:
    raise RuntimeError(
        "origin/main no longer matches the frozen Stage26-1 commit."
    )


if STATUS:
    raise RuntimeError(
        "Repository must be clean before Stage26-2:\n"
        + STATUS
    )


for path in [
    PROTOCOL_PATH,
    EXECUTION_PLAN_PATH,
    PREFLIGHT_RECEIPT_PATH,
    COLD_RECEIPT_PATH,
]:

    if not path.exists():
        raise FileNotFoundError(
            path
        )


immutable_checks = [
    (
        "measurement_protocol",
        PROTOCOL_PATH,
        EXPECTED_PROTOCOL_SHA256,
    ),
    (
        "cpu_execution_plan",
        EXECUTION_PLAN_PATH,
        EXPECTED_EXECUTION_PLAN_SHA256,
    ),
    (
        "cpu_preflight_receipt",
        PREFLIGHT_RECEIPT_PATH,
        EXPECTED_PREFLIGHT_RECEIPT_SHA256,
    ),
    (
        "cold_start_receipt",
        COLD_RECEIPT_PATH,
        EXPECTED_COLD_RECEIPT_SHA256,
    ),
]


for name, path, expected_sha in immutable_checks:

    actual_sha = sha256_file(
        path
    )

    ok = (
        actual_sha
        ==
        expected_sha
    )

    print(
        f"{name:28s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual_sha}"
    )

    if not ok:
        raise RuntimeError(
            f"Upstream frozen artifact changed: {name}"
        )


# =============================================================================
# 4. FINAL-PACKAGE DOUBLE-RUN GATE
# =============================================================================

banner(
    "STAGE26-2 :: RESUME / DOUBLE-RUN GATE"
)


if REPO_STAGE26_2_DIR.exists():

    raise RuntimeError(
        "Stage26-2 repository package already exists. "
        "Do NOT remeasure completed Stage26-2."
    )


existing_receipts = sorted(
    CONDITION_DIR.glob(
        "condition_*.json"
    )
)


print(
    "Previously durable condition receipts:",
    len(
        existing_receipts
    ),
)


if existing_receipts:

    print(
        "[RESUME MODE] Existing completed conditions will NOT be remeasured."
    )

else:

    print(
        "[FRESH MODE] No Stage26-2 condition has yet been measured."
    )


# If a prior methodological/environment failure exists, do not silently resume.
failure_markers = [
    STEADY_DIR
    / "stage26_2_invalid_environment.json",

    STEADY_DIR
    / "stage26_2_worker_failure.json",
]


for marker in failure_markers:

    if marker.exists():

        print(
            "\nExisting failure marker:"
        )

        print(
            marker
        )

        raise RuntimeError(
            "Stage26-2 previously stopped at a non-resource failure. "
            "Do not silently resume; send the failure record for a scoped fix."
        )


# =============================================================================
# 5. LOAD AND VERIFY EXACT FROZEN 80-CONDITION PLAN
# =============================================================================

banner(
    "STAGE26-2 :: LOAD FROZEN CPU EXECUTION PLAN"
)


plan = json.loads(
    EXECUTION_PLAN_PATH.read_text(
        encoding="utf-8"
    )
)


if plan.get(
    "condition_count"
) != 80:

    raise RuntimeError(
        "Expected exactly 80 frozen CPU conditions."
    )


conditions = sorted(
    plan[
        "conditions"
    ],
    key=lambda row: int(
        row[
            "execution_order"
        ]
    ),
)


if [
    int(
        row[
            "execution_order"
        ]
    )
    for row in conditions
] != list(
    range(
        1,
        81,
    )
):

    raise RuntimeError(
        "Frozen CPU execution orders are not exactly 1..80."
    )


expected_targets = {
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
    "ENS_LGBM_XGB_EQUAL",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
}


if {
    row[
        "target_id"
    ]
    for row in conditions
} != expected_targets:

    raise RuntimeError(
        "Frozen target universe does not match Stage26 protocol."
    )


print(
    "Frozen conditions:",
    len(
        conditions
    ),
)

print(
    "Frozen targets   :",
    len(
        expected_targets
    ),
)


print(
    "\nFirst 15 execution orders:"
)


for row in conditions[:15]:

    print(
        f"{row['execution_order']:03d} "
        f"{row['target_id']:43s} "
        f"{row['hardware_mode']:21s} "
        f"B={int(row['batch_size']):5d} "
        f"W={int(row['warmup_runs']):3d} "
        f"T={int(row['timed_runs']):3d}"
    )


# =============================================================================
# 6. LOAD BATCH-1 PREDICTION INTEGRITY REFERENCES
# =============================================================================

banner(
    "STAGE26-2 :: BATCH-1 PREDICTION INTEGRITY REFERENCES"
)


preflight = json.loads(
    PREFLIGHT_RECEIPT_PATH.read_text(
        encoding="utf-8"
    )
)


expected_b1_fingerprints = {}


for row in preflight[
    "results"
]:

    key = (
        row[
            "target_id"
        ],
        row[
            "hardware_mode"
        ],
    )

    expected_b1_fingerprints[
        key
    ] = row[
        "output_sha256"
    ]


expected_b1_keys = {
    (
        target,
        mode,
    )
    for target in expected_targets
    for mode in [
        "CPU_1_PHYSICAL_CORE",
        "CPU_2_PHYSICAL_CORE",
    ]
}


if set(
    expected_b1_fingerprints
) != expected_b1_keys:

    raise RuntimeError(
        "Could not reconstruct all 16 frozen batch-1 fingerprints."
    )


print(
    "Frozen B=1 fingerprints:",
    len(
        expected_b1_fingerprints
    ),
)


# =============================================================================
# 7. CREATE WARM-INFERENCE WORKER
# =============================================================================

banner(
    "STAGE26-2 :: CREATE WARM CPU WORKER"
)


worker_source = r'''
from __future__ import annotations

import os
import sys
import gc
import json
import hashlib
import traceback
import importlib.util
from pathlib import Path
from time import perf_counter_ns


def atomic_json(path, obj):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    raw = (
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
        )
        + "\n"
    ).encode(
        "utf-8"
    )

    with tmp.open(
        "wb"
    ) as f:

        f.write(
            raw
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def import_path(
    name,
    path,
):

    spec = (
        importlib.util.spec_from_file_location(
            name,
            str(
                path
            ),
        )
    )

    if (
        spec is None
        or
        spec.loader is None
    ):

        raise RuntimeError(
            f"Unable to import {path}"
        )

    module = (
        importlib.util.module_from_spec(
            spec
        )
    )

    spec.loader.exec_module(
        module
    )

    return module


def sha256_array(array):

    import numpy as np

    a = np.ascontiguousarray(
        array
    )

    h = hashlib.sha256()

    h.update(
        str(
            a.dtype
        ).encode(
            "ascii"
        )
    )

    h.update(
        json.dumps(
            list(
                a.shape
            )
        ).encode(
            "ascii"
        )
    )

    h.update(
        a.tobytes(
            order="C"
        )
    )

    return h.hexdigest()


def extract_torch_state(obj):

    import torch

    if not isinstance(
        obj,
        dict,
    ):

        raise TypeError(
            f"Unsupported checkpoint root type: {type(obj)}"
        )

    if (
        obj
        and
        all(
            torch.is_tensor(
                value
            )
            for value in obj.values()
        )
    ):

        return obj

    for key in [
        "model_state_dict",
        "state_dict",
        "model_state",
        "network_state_dict",
        "model",
    ]:

        candidate = obj.get(
            key
        )

        if (
            isinstance(
                candidate,
                dict,
            )
            and
            candidate
            and
            all(
                torch.is_tensor(
                    value
                )
                for value in candidate.values()
            )
        ):

            return candidate

    raise RuntimeError(
        "Unable to identify torch state_dict."
    )


def torch_load_state(path):

    import torch

    try:

        obj = torch.load(
            str(
                path
            ),
            map_location="cpu",
            weights_only=True,
        )

    except TypeError:

        obj = torch.load(
            str(
                path
            ),
            map_location="cpu",
        )

    return extract_torch_state(
        obj
    )


def configure_torch_threads(
    thread_count,
):

    import torch

    torch.set_num_threads(
        int(
            thread_count
        )
    )

    try:

        torch.set_num_interop_threads(
            1
        )

    except RuntimeError:

        pass

    if int(
        torch.get_num_threads()
    ) != int(
        thread_count
    ):

        raise RuntimeError(
            "PyTorch intra-op thread mismatch."
        )

    if int(
        torch.get_num_interop_threads()
    ) != 1:

        raise RuntimeError(
            "PyTorch inter-op thread mismatch."
        )


def validate_probability_vector(
    probability,
    *,
    expected_rows,
):

    import numpy as np

    p = np.asarray(
        probability
    ).reshape(
        -1
    )

    if p.shape != (
        int(
            expected_rows
        ),
    ):

        raise RuntimeError(
            f"Unexpected output shape: {p.shape}; "
            f"expected {(int(expected_rows),)}"
        )

    if not np.isfinite(
        p
    ).all():

        raise RuntimeError(
            "Non-finite probability."
        )

    if (
        (p < 0.0).any()
        or
        (p > 1.0).any()
    ):

        raise RuntimeError(
            "Probability outside [0,1]."
        )

    return p


# =============================================================================
# GROUP A INPUT
# =============================================================================

def build_group_a_input(
    *,
    repo,
    batch_size,
    seed,
):

    import joblib
    import numpy as np

    scaler = joblib.load(
        repo
        / "results/stage15_transformer_checkpoint/"
          "stage15_2_standard_scaler.joblib"
    )

    if int(
        scaler.n_features_in_
    ) != 70:

        raise RuntimeError(
            "Frozen scaler feature count mismatch."
        )

    derived_seed = (
        int(
            seed
        )
        +
        int(
            batch_size
        )
    )

    rng = np.random.default_rng(
        derived_seed
    )

    Z = rng.normal(
        loc=0.0,
        scale=1.0,
        size=(
            int(
                batch_size
            ),
            70,
        ),
    ).astype(
        np.float32
    )

    mean = np.asarray(
        scaler.mean_,
        dtype=np.float32,
    )

    scale = np.asarray(
        scaler.scale_,
        dtype=np.float32,
    )

    X_raw = (
        mean[
            None,
            :
        ]
        +
        Z
        *
        scale[
            None,
            :
        ]
    ).astype(
        np.float32,
        copy=False,
    )

    if not np.isfinite(
        Z
    ).all():

        raise RuntimeError(
            "Non-finite FT synthetic input."
        )

    if not np.isfinite(
        X_raw
    ).all():

        raise RuntimeError(
            "Non-finite tree synthetic input."
        )

    return (
        Z,
        X_raw,
        derived_seed,
    )


# =============================================================================
# GROUP B INPUT
# =============================================================================

def make_ipv4_packet(
    rng,
    protocol,
    length,
):

    import numpy as np

    packet = bytearray(
        rng.integers(
            0,
            256,
            size=int(
                length
            ),
            dtype=np.uint8,
        ).tobytes()
    )

    packet[
        0
    ] = 0x45

    packet[
        2:4
    ] = int(
        length
    ).to_bytes(
        2,
        "big",
    )

    packet[
        6
    ] = (
        packet[
            6
        ]
        &
        0xE0
    )

    packet[
        7
    ] = 0

    packet[
        9
    ] = int(
        protocol
    )

    return bytes(
        packet
    )


def build_packet_input(
    *,
    repo,
    batch_size,
    seed,
    need_scaled_float,
):

    import numpy as np

    encoder = import_path(
        "stage26_packet_encoder",
        (
            repo
            / "scripts/stage20_packet_image_encoder.py"
        ),
    )

    derived_seed = (
        int(
            seed
        )
        +
        1_000_000
        +
        int(
            batch_size
        )
    )

    rng = np.random.default_rng(
        derived_seed
    )

    images_uint8 = np.zeros(
        (
            int(
                batch_size
            ),
            1,
            64,
            256,
        ),
        dtype=np.uint8,
    )

    padding_mask = np.zeros(
        (
            int(
                batch_size
            ),
            1,
            64,
            256,
        ),
        dtype=np.bool_,
    )

    for flow_index in range(
        int(
            batch_size
        )
    ):

        packet_count = int(
            rng.integers(
                1,
                65,
            )
        )

        packets = []

        for _ in range(
            packet_count
        ):

            protocol = (
                6
                if int(
                    rng.integers(
                        0,
                        2,
                    )
                ) == 0
                else 17
            )

            packet_length = int(
                rng.integers(
                    40,
                    257,
                )
            )

            packets.append(
                make_ipv4_packet(
                    rng,
                    protocol,
                    packet_length,
                )
            )

        image, mask = (
            encoder.encode_flow(
                packets
            )
        )

        images_uint8[
            flow_index,
            0,
        ] = image

        padding_mask[
            flow_index,
            0,
        ] = mask


    if need_scaled_float:

        image_model = (
            images_uint8.astype(
                np.float32
            )
            /
            np.float32(
                255.0
            )
        )

        del images_uint8

        return (
            image_model,
            padding_mask,
            derived_seed,
        )


    return (
        images_uint8,
        padding_mask,
        derived_seed,
    )


# =============================================================================
# FT CONSTRUCTION
# =============================================================================

def build_ft_model(
    *,
    ft_module,
    architecture_record,
    checkpoint,
):

    arch = architecture_record[
        "architecture"
    ]

    model = (
        ft_module.NumericFTTransformer(
            n_features=int(
                architecture_record[
                    "input_predictor_count"
                ]
            ),
            d_token=int(
                arch[
                    "d_token"
                ]
            ),
            n_heads=int(
                arch[
                    "n_heads"
                ]
            ),
            n_layers=int(
                arch[
                    "n_layers"
                ]
            ),
            d_ff=int(
                arch[
                    "d_ff"
                ]
            ),
            dropout=float(
                arch[
                    "dropout"
                ]
            ),
        )
    )

    state = torch_load_state(
        checkpoint
    )

    model.load_state_dict(
        state,
        strict=True,
    )

    model.eval()

    parameter_count = int(
        sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        )
    )

    if parameter_count != 159169:

        raise RuntimeError(
            f"FT parameter count mismatch: {parameter_count}"
        )

    return model


# =============================================================================
# MEASUREMENT
# =============================================================================

def execute_measurement(
    config,
):

    repo = Path(
        config[
            "repo"
        ]
    )

    target = config[
        "target_id"
    ]

    hardware_mode = config[
        "hardware_mode"
    ]

    thread_count = int(
        config[
            "thread_count"
        ]
    )

    affinity = [
        int(
            x
        )
        for x in config[
            "affinity"
        ]
    ]

    batch_size = int(
        config[
            "batch_size"
        ]
    )

    warmup_runs = int(
        config[
            "warmup_runs"
        ]
    )

    timed_runs = int(
        config[
            "timed_runs"
        ]
    )

    seed = int(
        config[
            "measurement_seed"
        ]
    )

    condition_id = config[
        "condition_id"
    ]

    execution_order = int(
        config[
            "execution_order"
        ]
    )


    # -------------------------------------------------------------------------
    # CPU affinity before numerical / ML imports.
    # -------------------------------------------------------------------------

    if hasattr(
        os,
        "sched_setaffinity",
    ):

        os.sched_setaffinity(
            0,
            set(
                affinity
            ),
        )


    observed_affinity = (
        sorted(
            os.sched_getaffinity(
                0
            )
        )
        if hasattr(
            os,
            "sched_getaffinity",
        )
        else
        None
    )


    if (
        observed_affinity is not None
        and
        observed_affinity != affinity
    ):

        raise RuntimeError(
            f"Affinity mismatch: "
            f"{observed_affinity} != {affinity}"
        )


    # Default CPython GC semantics are preserved.
    gc_enabled = bool(
        gc.isenabled()
    )


    # -------------------------------------------------------------------------
    # Imports + input + model load OUTSIDE timed region.
    # -------------------------------------------------------------------------

    if target in {
        "STAGE16_XGBOOST_TUNED",
        "STAGE16_LIGHTGBM_TUNED",
        "STAGE16_CATBOOST_TUNED",
        "ENS_LGBM_XGB_EQUAL",
    }:

        import numpy as np
        import joblib

        Z, X_raw, derived_seed = (
            build_group_a_input(
                repo=repo,
                batch_size=batch_size,
                seed=seed,
            )
        )


        if target == "STAGE16_XGBOOST_TUNED":

            import xgboost

            model = joblib.load(
                repo
                / "results/stage16_classical_benchmark_checkpoint/"
                  "stage16_3_tuned_models/XGBOOST_tuned.joblib"
            )

            model.set_params(
                n_jobs=thread_count
            )


            def infer():

                return model.predict_proba(
                    X_raw
                )[:, 1]


        elif target == "STAGE16_LIGHTGBM_TUNED":

            import lightgbm

            model = joblib.load(
                repo
                / "results/stage16_classical_benchmark_checkpoint/"
                  "stage16_3_tuned_models/LIGHTGBM_tuned.joblib"
            )

            model.set_params(
                n_jobs=thread_count
            )


            def infer():

                return model.predict_proba(
                    X_raw
                )[:, 1]


        elif target == "STAGE16_CATBOOST_TUNED":

            import catboost

            model = joblib.load(
                repo
                / "results/stage16_classical_benchmark_checkpoint/"
                  "stage16_3_tuned_models/CATBOOST_tuned.joblib"
            )


            def infer():

                return model.predict_proba(
                    X_raw,
                    thread_count=thread_count,
                )[:, 1]


        else:

            import xgboost
            import lightgbm

            xgb_model = joblib.load(
                repo
                / "results/stage16_classical_benchmark_checkpoint/"
                  "stage16_3_tuned_models/XGBOOST_tuned.joblib"
            )

            lgb_model = joblib.load(
                repo
                / "results/stage16_classical_benchmark_checkpoint/"
                  "stage16_3_tuned_models/LIGHTGBM_tuned.joblib"
            )

            xgb_model.set_params(
                n_jobs=thread_count
            )

            lgb_model.set_params(
                n_jobs=thread_count
            )


            def infer():

                xgb_p = (
                    xgb_model.predict_proba(
                        X_raw
                    )[:, 1]
                )

                lgb_p = (
                    lgb_model.predict_proba(
                        X_raw
                    )[:, 1]
                )

                return (
                    np.asarray(
                        xgb_p,
                        dtype=np.float64,
                    )
                    +
                    np.asarray(
                        lgb_p,
                        dtype=np.float64,
                    )
                ) / 2.0


    elif target in {
        "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
        "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
    }:

        import numpy as np
        import torch

        configure_torch_threads(
            thread_count
        )

        derived_seed = (
            seed
            +
            batch_size
        )

        rng = np.random.default_rng(
            derived_seed
        )

        Z = rng.normal(
            0.0,
            1.0,
            size=(
                batch_size,
                70,
            ),
        ).astype(
            np.float32
        )

        x = torch.from_numpy(
            Z
        )

        ft_module = import_path(
            "stage26_ft_module",
            (
                repo
                / "results/stage15_transformer_checkpoint/"
                  "ft_transformer_numeric.py"
            ),
        )

        architecture_record = json.loads(
            (
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4c_frozen_architecture.json"
            ).read_text(
                encoding="utf-8"
            )
        )

        checkpoint_paths = {
            7:
                (
                    repo
                    / "results/stage15_transformer_checkpoint/"
                      "stage15_4b_models/"
                      "FT_BALANCED_seed_7_best_extended.pt"
                ),

            29:
                (
                    repo
                    / "results/stage15_transformer_checkpoint/"
                      "stage15_4a_models/"
                      "FT_BALANCED_seed_29_best.pt"
                ),

            101:
                (
                    repo
                    / "results/stage15_transformer_checkpoint/"
                      "stage15_4a_models/"
                      "FT_BALANCED_seed_101_best.pt"
                ),

            313:
                (
                    repo
                    / "results/stage15_transformer_checkpoint/"
                      "stage15_4c_models/"
                      "FT_BALANCED_seed_313_best.pt"
                ),

            997:
                (
                    repo
                    / "results/stage15_transformer_checkpoint/"
                      "stage15_4c_models/"
                      "FT_BALANCED_seed_997_best.pt"
                ),
        }

        seeds = (
            [
                7,
                29,
                101,
                313,
                997,
            ]
            if
            target
            ==
            "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING"
            else
            [
                7
            ]
        )

        models = []

        for checkpoint_seed in seeds:

            models.append(
                build_ft_model(
                    ft_module=ft_module,
                    architecture_record=architecture_record,
                    checkpoint=checkpoint_paths[
                        checkpoint_seed
                    ],
                )
            )


        def infer():

            member_probabilities = []

            with torch.inference_mode():

                for model in models:

                    logits = model(
                        x
                    )

                    member_probabilities.append(
                        torch.sigmoid(
                            logits
                        )
                        .detach()
                        .cpu()
                        .numpy()
                    )

            return np.mean(
                np.stack(
                    member_probabilities,
                    axis=0,
                ),
                axis=0,
            )


    elif target == "STAGE20_MASKED_CNN_V1":

        import numpy as np
        import torch

        configure_torch_threads(
            thread_count
        )

        (
            image_np,
            mask_np,
            derived_seed,
        ) = build_packet_input(
            repo=repo,
            batch_size=batch_size,
            seed=seed,
            need_scaled_float=False,
        )

        cnn_module = import_path(
            "stage26_cnn_module",
            (
                repo
                / "scripts/stage20_masked_cnn.py"
            ),
        )

        model = (
            cnn_module.Stage20MaskedCNNv1()
        )

        state = torch_load_state(
            repo
            / "results/stage20_1e_training/"
              "stage20_1e2_epoch10_model_state_dict.pt"
        )

        model.load_state_dict(
            state,
            strict=True,
        )

        model.eval()

        parameter_count = int(
            cnn_module.count_trainable_parameters(
                model
            )
        )

        if parameter_count != 93025:

            raise RuntimeError(
                f"CNN parameter mismatch: {parameter_count}"
            )

        # Authentic frozen CNN boundary:
        # uint8 image, bool mask; float32/255 occurs inside forward().
        image = torch.from_numpy(
            image_np
        )

        mask = torch.from_numpy(
            mask_np
        )


        def infer():

            with torch.inference_mode():

                logits = model(
                    image,
                    mask,
                )

                return (
                    torch.sigmoid(
                        logits
                    )
                    .detach()
                    .cpu()
                    .numpy()
                )


    elif target == "STAGE21_MASKED_VIT_V1":

        import numpy as np
        import torch

        configure_torch_threads(
            thread_count
        )

        (
            image_np,
            mask_np,
            derived_seed,
        ) = build_packet_input(
            repo=repo,
            batch_size=batch_size,
            seed=seed,
            need_scaled_float=True,
        )

        vit_module = import_path(
            "stage26_vit_module",
            (
                repo
                / "scripts/stage21_masked_vit.py"
            ),
        )

        model = (
            vit_module.Stage21MaskedViTv1()
        )

        state = torch_load_state(
            repo
            / "results/stage21_architecture/"
              "stage21_2_epoch10_model_state_dict.pt"
        )

        model.load_state_dict(
            state,
            strict=True,
        )

        model.eval()

        parameter_count = int(
            vit_module.count_trainable_parameters(
                model
            )
        )

        if parameter_count != 91969:

            raise RuntimeError(
                f"ViT parameter mismatch: {parameter_count}"
            )

        # Authentic frozen ViT boundary:
        # already-scaled float32 image + bool mask.
        image = torch.from_numpy(
            image_np
        )

        mask = torch.from_numpy(
            mask_np
        )


        def infer():

            with torch.inference_mode():

                logits = model(
                    image,
                    mask,
                )

                return (
                    torch.sigmoid(
                        logits
                    )
                    .detach()
                    .cpu()
                    .numpy()
                )


    else:

        raise RuntimeError(
            f"Unknown target: {target}"
        )


    # -------------------------------------------------------------------------
    # Exact frozen warmup count.
    # -------------------------------------------------------------------------

    warmup_last_output = None

    for _ in range(
        warmup_runs
    ):

        warmup_last_output = infer()


    warmup_p = validate_probability_vector(
        warmup_last_output,
        expected_rows=batch_size,
    )

    warmup_output_sha256 = (
        sha256_array(
            warmup_p
        )
    )


    # -------------------------------------------------------------------------
    # Exact frozen timed count.
    # -------------------------------------------------------------------------

    elapsed_ns = []

    timed_last_output = None


    for _ in range(
        timed_runs
    ):

        start_ns = (
            perf_counter_ns()
        )

        timed_last_output = infer()

        end_ns = (
            perf_counter_ns()
        )

        elapsed = int(
            end_ns
            -
            start_ns
        )

        if elapsed <= 0:

            raise RuntimeError(
                f"Non-positive elapsed_ns: {elapsed}"
            )

        elapsed_ns.append(
            elapsed
        )


    timed_p = validate_probability_vector(
        timed_last_output,
        expected_rows=batch_size,
    )

    timed_output_sha256 = (
        sha256_array(
            timed_p
        )
    )


    if (
        warmup_output_sha256
        !=
        timed_output_sha256
    ):

        raise RuntimeError(
            "Prediction fingerprint changed between "
            "warmup and timed steady-state execution."
        )


    return {
        "schema":
            "stage26_2_condition_receipt_v1",

        "status":
            "PASS",

        "condition_id":
            condition_id,

        "execution_order":
            execution_order,

        "target_id":
            target,

        "hardware_mode":
            hardware_mode,

        "thread_count":
            thread_count,

        "affinity_requested":
            affinity,

        "affinity_observed":
            observed_affinity,

        "batch_size":
            batch_size,

        "warmup_runs":
            warmup_runs,

        "timed_runs":
            timed_runs,

        "derived_input_seed":
            int(
                derived_seed
            ),

        "python_gc_enabled":
            gc_enabled,

        "elapsed_ns":
            elapsed_ns,

        "warmup_output_sha256":
            warmup_output_sha256,

        "timed_output_sha256":
            timed_output_sha256,

        "timing_clock":
            "time.perf_counter_ns",

        "timing_boundary":
            (
                "prepared_model_input_to_materialized_attack_probability"
            ),

        "memory_profiled":
            False,

        "holdout_accessed":
            False,

        "gpu_used":
            False,
    }


def main():

    config_path = Path(
        sys.argv[
            1
        ]
    )

    config = json.loads(
        config_path.read_text(
            encoding="utf-8"
        )
    )

    result_path = Path(
        config[
            "result_path"
        ]
    )


    try:

        result = execute_measurement(
            config
        )


    except MemoryError as exc:

        result = {
            "schema":
                "stage26_2_condition_receipt_v1",

            "status":
                "RESOURCE_LIMIT_OOM",

            "condition_id":
                config[
                    "condition_id"
                ],

            "execution_order":
                int(
                    config[
                        "execution_order"
                    ]
                ),

            "target_id":
                config[
                    "target_id"
                ],

            "hardware_mode":
                config[
                    "hardware_mode"
                ],

            "thread_count":
                int(
                    config[
                        "thread_count"
                    ]
                ),

            "affinity_requested":
                config[
                    "affinity"
                ],

            "batch_size":
                int(
                    config[
                        "batch_size"
                    ]
                ),

            "warmup_runs":
                int(
                    config[
                        "warmup_runs"
                    ]
                ),

            "timed_runs":
                int(
                    config[
                        "timed_runs"
                    ]
                ),

            "exception_type":
                type(
                    exc
                ).__name__,

            "exception_message":
                str(
                    exc
                ),

            "elapsed_ns":
                [],

            "memory_profiled":
                False,

            "holdout_accessed":
                False,

            "gpu_used":
                False,
        }


    except RuntimeError as exc:

        message = str(
            exc
        ).lower()

        oom_like = (
            "out of memory"
            in
            message
            or
            "cannot allocate memory"
            in
            message
            or
            "bad alloc"
            in
            message
        )


        if oom_like:

            result = {
                "schema":
                    "stage26_2_condition_receipt_v1",

                "status":
                    "RESOURCE_LIMIT_OOM",

                "condition_id":
                    config[
                        "condition_id"
                    ],

                "execution_order":
                    int(
                        config[
                            "execution_order"
                        ]
                    ),

                "target_id":
                    config[
                        "target_id"
                    ],

                "hardware_mode":
                    config[
                        "hardware_mode"
                    ],

                "thread_count":
                    int(
                        config[
                            "thread_count"
                        ]
                    ),

                "affinity_requested":
                    config[
                        "affinity"
                    ],

                "batch_size":
                    int(
                        config[
                            "batch_size"
                        ]
                    ),

                "warmup_runs":
                    int(
                        config[
                            "warmup_runs"
                        ]
                    ),

                "timed_runs":
                    int(
                        config[
                            "timed_runs"
                        ]
                    ),

                "exception_type":
                    type(
                        exc
                    ).__name__,

                "exception_message":
                    str(
                        exc
                    ),

                "elapsed_ns":
                    [],

                "memory_profiled":
                    False,

                "holdout_accessed":
                    False,

                "gpu_used":
                    False,
            }


        else:

            result = {
                "schema":
                    "stage26_2_condition_receipt_v1",

                "status":
                    "WORKER_FAILURE",

                "condition_id":
                    config[
                        "condition_id"
                    ],

                "execution_order":
                    int(
                        config[
                            "execution_order"
                    ]
                ),

                "target_id":
                    config[
                        "target_id"
                    ],

                "hardware_mode":
                    config[
                        "hardware_mode"
                    ],

                "thread_count":
                    int(
                        config[
                            "thread_count"
                    ]
                ),

                "affinity_requested":
                    config[
                        "affinity"
                    ],

                "batch_size":
                    int(
                        config[
                            "batch_size"
                    ]
                ),

                "warmup_runs":
                    int(
                        config[
                            "warmup_runs"
                    ]
                ),

                "timed_runs":
                    int(
                        config[
                            "timed_runs"
                    ]
                ),

                "exception_type":
                    type(
                        exc
                    ).__name__,

                "exception_message":
                    str(
                        exc
                    ),

                "traceback":
                    traceback.format_exc(),

                "elapsed_ns":
                    [],

                "memory_profiled":
                    False,

                "holdout_accessed":
                    False,

                "gpu_used":
                    False,
            }


    except Exception as exc:

        result = {
            "schema":
                "stage26_2_condition_receipt_v1",

            "status":
                "WORKER_FAILURE",

            "condition_id":
                config[
                    "condition_id"
                ],

            "execution_order":
                int(
                    config[
                        "execution_order"
                    ]
                ),

            "target_id":
                config[
                    "target_id"
                ],

            "hardware_mode":
                config[
                    "hardware_mode"
                ],

            "thread_count":
                int(
                    config[
                        "thread_count"
                    ]
                ),

            "affinity_requested":
                config[
                    "affinity"
                ],

            "batch_size":
                int(
                    config[
                        "batch_size"
                    ]
                ),

            "warmup_runs":
                int(
                    config[
                        "warmup_runs"
                    ]
                ),

            "timed_runs":
                int(
                    config[
                        "timed_runs"
                    ]
                ),

            "exception_type":
                type(
                    exc
                ).__name__,

            "exception_message":
                str(
                    exc
                ),

            "traceback":
                traceback.format_exc(),

            "elapsed_ns":
                [],

            "memory_profiled":
                False,

            "holdout_accessed":
                False,

            "gpu_used":
                False,
        }


    atomic_json(
        result_path,
        result,
    )


    # One machine-readable line last.
    print(
        json.dumps(
            {
                "status":
                    result[
                        "status"
                    ],

                "condition_id":
                    result[
                        "condition_id"
                    ],

                "execution_order":
                    result[
                        "execution_order"
                    ],

                "target_id":
                    result[
                        "target_id"
                    ],

                "hardware_mode":
                    result[
                        "hardware_mode"
                    ],

                "batch_size":
                    result[
                        "batch_size"
                    ],
            },
            sort_keys=True,
        ),
        flush=True,
    )


    if result[
        "status"
    ] == "WORKER_FAILURE":

        sys.exit(
            2
        )


if __name__ == "__main__":

    main()
'''


WORKER_PATH.write_text(
    textwrap.dedent(
        worker_source
    ).lstrip(),
    encoding="utf-8",
)


worker_sha = sha256_file(
    WORKER_PATH
)


print(
    "Worker:"
)

print(
    " ",
    WORKER_PATH,
)

print(
    "SHA256:"
)

print(
    " ",
    worker_sha,
)


# =============================================================================
# 8. FREEZE STAGE26-2 IMPLEMENTATION BEFORE FIRST NEW CONDITION
# =============================================================================

banner(
    "STAGE26-2 :: IMPLEMENTATION FREEZE"
)


implementation = {
    "schema":
        "stage26_2_warm_cpu_implementation_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-2",

    "parent_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "cpu_execution_plan_sha256":
        EXPECTED_EXECUTION_PLAN_SHA256,

    "cpu_preflight_receipt_sha256":
        EXPECTED_PREFLIGHT_RECEIPT_SHA256,

    "cold_start_receipt_sha256":
        EXPECTED_COLD_RECEIPT_SHA256,

    "worker_sha256":
        worker_sha,

    "execution_plan":
        "EXACT_FROZEN_80_CONDITION_PLAN",

    "condition_isolation":
        "ONE_FRESH_PYTHON_SUBPROCESS_PER_MODEL_HARDWARE_BATCH_CONDITION",

    "thread_environment_set_before_worker_import":
        True,

    "cpu_affinity_set_before_ml_import":
        True,

    "input_generation":
        {
            "GROUP_A":
                "derived_seed = 26042 + batch_size",

            "GROUP_B":
                "derived_seed = 26042 + 1000000 + batch_size",
        },

    "timing_clock":
        "time.perf_counter_ns",

    "timing_boundary":
        "PREPARED_MODEL_INPUT_TO_MATERIALIZED_ATTACK_PROBABILITY",

    "cnn_boundary":
        (
            "uint8 image + bool padding mask; CNN float32 conversion "
            "and /255 remain inside frozen forward and therefore inside T_infer"
        ),

    "vit_boundary":
        (
            "already-scaled float32 image + bool padding mask; scaling "
            "occurs before T_infer"
        ),

    "ft_ensemble_boundary":
        (
            "all five checkpoint forwards plus arithmetic probability "
            "averaging are inside T_infer"
        ),

    "classical_ensemble_boundary":
        (
            "XGBoost probability + LightGBM probability + equal arithmetic "
            "averaging are inside T_infer"
        ),

    "warmup_and_timed_counts":
        {
            "1":
                {
                    "warmup":
                        50,
                    "timed":
                        200,
                },

            "64":
                {
                    "warmup":
                        30,
                    "timed":
                        150,
                },

            "256":
                {
                    "warmup":
                        20,
                    "timed":
                        100,
                },

            "1024":
                {
                    "warmup":
                        10,
                    "timed":
                        50,
                },

            "8192":
                {
                    "warmup":
                        5,
                    "timed":
                        20,
                },
        },

    "environment_gate":
        {
            "cpu_percent_max":
                CPU_UTILIZATION_GATE_PERCENT,

            "sample_seconds":
                CPU_UTILIZATION_SAMPLE_SECONDS,

            "available_ram_gib_min":
                MIN_AVAILABLE_RAM_GIB,

            "retry_count":
                ENVIRONMENT_RETRY_COUNT,

            "retry_cooldown_seconds":
                ENVIRONMENT_RETRY_COOLDOWN_SECONDS,
        },

    "condition_cooldown_seconds":
        CONDITION_COOLDOWN_SECONDS,

    "condition_timeout_seconds":
        CONDITION_TIMEOUT_SECONDS,

    "oom_policy":
        "RECORD_RESOURCE_LIMIT_OOM_AND_DO_NOT_CHANGE_BATCH",

    "timeout_policy":
        "RECORD_TIMEOUT_RESOURCE_LIMIT_AND_DO_NOT_CHANGE_POLICY",

    "python_gc_policy":
        "DEFAULT_ENABLED_NO_FORCED_COLLECTION_INSIDE_TIMED_LOOP",

    "raw_nanosecond_retention":
        True,

    "resume_rule":
        (
            "Existing final condition receipts are never remeasured; "
            "resume continues only unexecuted execution orders."
        ),

    "memory_profiling":
        False,

    "holdout_access":
        False,

    "gpu":
        False,
}


if IMPLEMENTATION_PATH.exists():

    existing_implementation = json.loads(
        IMPLEMENTATION_PATH.read_text(
            encoding="utf-8"
        )
    )

    if existing_implementation != implementation:

        raise RuntimeError(
            "Existing Stage26-2 implementation record differs. "
            "Do not silently mutate measurement implementation."
        )

else:

    write_json(
        IMPLEMENTATION_PATH,
        implementation,
    )


implementation_sha = sha256_file(
    IMPLEMENTATION_PATH
)


print(
    "Implementation SHA256:"
)

print(
    " ",
    implementation_sha,
)


# =============================================================================
# 9. ENVIRONMENT GATE
# =============================================================================

def environment_gate():

    attempts = []

    for attempt in range(
        1,
        ENVIRONMENT_RETRY_COUNT
        +
        1,
    ):

        cpu_percent = float(
            psutil.cpu_percent(
                interval=CPU_UTILIZATION_SAMPLE_SECONDS
            )
        )

        available_ram_gib = float(
            psutil.virtual_memory().available
            /
            1024**3
        )

        passed = (
            cpu_percent
            <=
            CPU_UTILIZATION_GATE_PERCENT
            and
            available_ram_gib
            >=
            MIN_AVAILABLE_RAM_GIB
        )

        record = {
            "attempt":
                attempt,

            "cpu_percent":
                cpu_percent,

            "available_ram_gib":
                available_ram_gib,

            "passed":
                passed,
        }

        attempts.append(
            record
        )

        if passed:

            return (
                True,
                attempts,
                record,
            )


        if attempt < ENVIRONMENT_RETRY_COUNT:

            time.sleep(
                ENVIRONMENT_RETRY_COOLDOWN_SECONDS
            )


    return (
        False,
        attempts,
        attempts[
            -1
        ],
    )


# =============================================================================
# 10. VALIDATE ANY EXISTING CONDITION RECEIPTS BEFORE RESUME
# =============================================================================

banner(
    "STAGE26-2 :: EXISTING RECEIPT VALIDATION"
)


condition_by_order = {
    int(
        row[
            "execution_order"
        ]
    ):
        row
    for row in conditions
}


existing_final_orders = set()


for receipt_path in sorted(
    CONDITION_DIR.glob(
        "condition_*.json"
    )
):

    receipt = json.loads(
        receipt_path.read_text(
            encoding="utf-8"
        )
    )

    order = int(
        receipt[
            "execution_order"
        ]
    )

    if order not in condition_by_order:

        raise RuntimeError(
            f"Unknown persisted execution order: {order}"
        )


    frozen = condition_by_order[
        order
    ]


    identity_fields = [
        "condition_id",
        "target_id",
        "hardware_mode",
        "thread_count",
        "batch_size",
        "warmup_runs",
        "timed_runs",
    ]


    for field in identity_fields:

        if receipt.get(
            field
        ) != frozen.get(
            field
        ):

            raise RuntimeError(
                f"Persisted condition {order} differs "
                f"from frozen plan at field {field}."
            )


    if receipt[
        "status"
    ] == "WORKER_FAILURE":

        raise RuntimeError(
            f"Persisted WORKER_FAILURE at execution order {order}. "
            "Do not silently resume."
        )


    if receipt[
        "status"
    ] not in ALLOWED_FINAL_CONDITION_STATUSES:

        raise RuntimeError(
            f"Unexpected persisted status at order {order}: "
            f"{receipt['status']}"
        )


    if receipt[
        "status"
    ] == "PASS":

        elapsed = receipt.get(
            "elapsed_ns",
            [],
        )

        if len(
            elapsed
        ) != int(
            frozen[
                "timed_runs"
            ]
        ):

            raise RuntimeError(
                f"Persisted PASS condition {order} has "
                "incorrect raw timing count."
            )


    existing_final_orders.add(
        order
    )


print(
    "Validated completed conditions:",
    len(
        existing_final_orders
    ),
)


# =============================================================================
# 11. EXECUTE FROZEN CONDITIONS IN FROZEN ORDER
# =============================================================================

banner(
    "STAGE26-2 :: EXECUTE FROZEN WARM CPU PLAN"
)


print(
    "Total frozen conditions:",
    len(
        conditions
    ),
)

print(
    "Already complete:",
    len(
        existing_final_orders
    ),
)

print(
    "GPU: OFF"
)

print(
    "Memory profiling: OFF"
)


new_condition_count = 0


for condition in conditions:

    execution_order = int(
        condition[
            "execution_order"
        ]
    )


    if execution_order in existing_final_orders:

        print(
            f"[SKIP] {execution_order:03d}/080 "
            f"already durably measured"
        )

        continue


    target = condition[
        "target_id"
    ]

    hardware_mode = condition[
        "hardware_mode"
    ]

    batch_size = int(
        condition[
            "batch_size"
        ]
    )

    threads = int(
        condition[
            "thread_count"
        ]
    )

    affinity = [
        int(
            x
        )
        for x in condition[
            "affinity"
        ]
    ]

    warmup_runs = int(
        condition[
            "warmup_runs"
        ]
    )

    timed_runs = int(
        condition[
            "timed_runs"
        ]
    )


    # Frozen cooldown between condition launches.
    if (
        execution_order > 1
        or
        new_condition_count > 0
    ):

        time.sleep(
            CONDITION_COOLDOWN_SECONDS
        )


    env_ok, env_attempts, env_final = (
        environment_gate()
    )


    if not env_ok:

        failure = {
            "schema":
                "stage26_2_invalid_environment_v1",

            "status":
                "INVALID_ENVIRONMENT",

            "execution_order":
                execution_order,

            "condition_id":
                condition[
                    "condition_id"
                ],

            "target_id":
                target,

            "hardware_mode":
                hardware_mode,

            "batch_size":
                batch_size,

            "environment_attempts":
                env_attempts,

            "completed_condition_orders":
                sorted(
                    existing_final_orders
                ),
        }


        marker = (
            STEADY_DIR
            / "stage26_2_invalid_environment.json"
        )


        write_json(
            marker,
            failure,
        )


        raise RuntimeError(
            "INVALID_ENVIRONMENT under frozen Stage26 rule. "
            "Do not silently continue or rerun."
        )


    result_path = (
        CONDITION_DIR
        / (
            f"condition_{execution_order:03d}.json"
        )
    )


    if result_path.exists():

        raise RuntimeError(
            f"Condition result appeared unexpectedly before launch: "
            f"{result_path}"
        )


    config = {
        "schema":
            "stage26_2_worker_config_v1",

        "repo":
            str(
                REPO_DIR
            ),

        "result_path":
            str(
                result_path
            ),

        "measurement_seed":
            MEASUREMENT_SEED,

        "condition_id":
            condition[
                "condition_id"
            ],

        "execution_order":
            execution_order,

        "target_id":
            target,

        "target_role":
            condition[
                "target_role"
            ],

        "comparison_group":
            condition[
                "comparison_group"
            ],

        "hardware_mode":
            hardware_mode,

        "thread_count":
            threads,

        "affinity":
            affinity,

        "batch_size":
            batch_size,

        "warmup_runs":
            warmup_runs,

        "timed_runs":
            timed_runs,
    }


    config_path = (
        CONFIG_DIR
        / (
            f"condition_{execution_order:03d}.json"
        )
    )


    write_json(
        config_path,
        config,
    )


    env = os.environ.copy()


    thread_value = str(
        threads
    )


    for key in [
        "OMP_NUM_THREADS",
        "MKL_NUM_THREADS",
        "OPENBLAS_NUM_THREADS",
        "NUMEXPR_NUM_THREADS",
        "BLIS_NUM_THREADS",
        "VECLIB_MAXIMUM_THREADS",
    ]:

        env[
            key
        ] = thread_value


    # Stage26 CPU phase must not accidentally expose CUDA.
    env[
        "CUDA_VISIBLE_DEVICES"
    ] = ""


    try:

        p = subprocess.run(
            [
                sys.executable,
                str(
                    WORKER_PATH
                ),
                str(
                    config_path
                ),
            ],
            cwd=REPO_DIR,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=False,
            timeout=CONDITION_TIMEOUT_SECONDS,
        )


    except subprocess.TimeoutExpired:

        timeout_receipt = {
            "schema":
                "stage26_2_condition_receipt_v1",

            "status":
                "TIMEOUT_RESOURCE_LIMIT",

            "condition_id":
                condition[
                    "condition_id"
                ],

            "execution_order":
                execution_order,

            "target_id":
                target,

            "target_role":
                condition[
                    "target_role"
                ],

            "comparison_group":
                condition[
                    "comparison_group"
                ],

            "hardware_mode":
                hardware_mode,

            "thread_count":
                threads,

            "affinity_requested":
                affinity,

            "batch_size":
                batch_size,

            "warmup_runs":
                warmup_runs,

            "timed_runs":
                timed_runs,

            "timeout_seconds":
                CONDITION_TIMEOUT_SECONDS,

            "environment_attempts":
                env_attempts,

            "environment_final":
                env_final,

            "elapsed_ns":
                [],

            "memory_profiled":
                False,

            "holdout_accessed":
                False,

            "gpu_used":
                False,
        }


        write_json(
            result_path,
            timeout_receipt,
        )


        existing_final_orders.add(
            execution_order
        )

        new_condition_count += 1


        print(
            f"[TIMEOUT] {execution_order:03d}/080 "
            f"{target:43s} "
            f"{hardware_mode:21s} "
            f"B={batch_size:5d}"
        )

        continue


    # -------------------------------------------------------------------------
    # Worker ended. Resolve receipt.
    # -------------------------------------------------------------------------

    if not result_path.exists():

        # A SIGKILL without the parent timeout is conservatively retained
        # as the frozen OOM/resource-limit category.
        if p.returncode in {
            -signal.SIGKILL,
            137,
        }:

            resource_receipt = {
                "schema":
                    "stage26_2_condition_receipt_v1",

                "status":
                    "RESOURCE_LIMIT_OOM",

                "resource_limit_reason":
                    (
                        "worker terminated by SIGKILL without parent timeout; "
                        "retained as resource-limit/OOM condition"
                    ),

                "condition_id":
                    condition[
                        "condition_id"
                    ],

                "execution_order":
                    execution_order,

                "target_id":
                    target,

                "target_role":
                    condition[
                        "target_role"
                    ],

                "comparison_group":
                    condition[
                        "comparison_group"
                    ],

                "hardware_mode":
                    hardware_mode,

                "thread_count":
                    threads,

                "affinity_requested":
                    affinity,

                "batch_size":
                    batch_size,

                "warmup_runs":
                    warmup_runs,

                "timed_runs":
                    timed_runs,

                "returncode":
                    p.returncode,

                "environment_attempts":
                    env_attempts,

                "environment_final":
                    env_final,

                "elapsed_ns":
                    [],

                "memory_profiled":
                    False,

                "holdout_accessed":
                    False,

                "gpu_used":
                    False,
            }


            write_json(
                result_path,
                resource_receipt,
            )


        else:

            failure = {
                "schema":
                    "stage26_2_worker_failure_v1",

                "status":
                    "WORKER_FAILURE",

                "condition_id":
                    condition[
                        "condition_id"
                    ],

                "execution_order":
                    execution_order,

                "target_id":
                    target,

                "hardware_mode":
                    hardware_mode,

                "batch_size":
                    batch_size,

                "returncode":
                    p.returncode,

                "stdout":
                    p.stdout[
                        -8000:
                    ],

                "stderr":
                    p.stderr[
                        -16000:
                    ],

                "environment_attempts":
                    env_attempts,
            }


            marker = (
                STEADY_DIR
                / "stage26_2_worker_failure.json"
            )


            write_json(
                marker,
                failure,
            )


            print(
                "\nWorker stdout:"
            )

            print(
                p.stdout
            )

            print(
                "\nWorker stderr:"
            )

            print(
                p.stderr
            )


            raise RuntimeError(
                "Stage26-2 worker exited without a durable condition receipt."
            )


    receipt = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


    # Add parent-side frozen environment receipt without modifying
    # the scientific timing observations themselves.
    receipt[
        "environment_attempts"
    ] = env_attempts

    receipt[
        "environment_final"
    ] = env_final

    receipt[
        "target_role"
    ] = condition[
        "target_role"
    ]

    receipt[
        "comparison_group"
    ] = condition[
        "comparison_group"
    ]


    write_json(
        result_path,
        receipt,
    )


    status = receipt[
        "status"
    ]


    if status == "WORKER_FAILURE":

        failure = {
            "schema":
                "stage26_2_worker_failure_v1",

            "status":
                "WORKER_FAILURE",

            "condition_receipt":
                receipt,
        }


        marker = (
            STEADY_DIR
            / "stage26_2_worker_failure.json"
        )


        write_json(
            marker,
            failure,
        )


        print(
            "\nFailure condition:"
        )

        print(
            json.dumps(
                receipt,
                indent=2,
                sort_keys=True,
            )
        )


        raise RuntimeError(
            "Stage26-2 WORKER_FAILURE. "
            "Stop for one scoped compatibility fix."
        )


    if status not in ALLOWED_FINAL_CONDITION_STATUSES:

        raise RuntimeError(
            f"Unexpected condition status: {status}"
        )


    # -------------------------------------------------------------------------
    # Frozen B=1 prediction-integrity gate.
    # -------------------------------------------------------------------------

    if (
        status == "PASS"
        and
        batch_size == 1
    ):

        expected_sha = (
            expected_b1_fingerprints[
                (
                    target,
                    hardware_mode,
                )
            ]
        )

        actual_sha = receipt[
            "timed_output_sha256"
        ]


        if actual_sha != expected_sha:

            failure = {
                "schema":
                    "stage26_2_batch1_prediction_integrity_failure_v1",

                "status":
                    "WORKER_FAILURE",

                "execution_order":
                    execution_order,

                "target_id":
                    target,

                "hardware_mode":
                    hardware_mode,

                "expected_sha256":
                    expected_sha,

                "actual_sha256":
                    actual_sha,
            }


            marker = (
                STEADY_DIR
                / "stage26_2_worker_failure.json"
            )


            write_json(
                marker,
                failure,
            )


            raise RuntimeError(
                "Stage26-2 B=1 prediction fingerprint changed "
                "from the frozen CPU preflight."
            )


    if status == "PASS":

        elapsed_ns = receipt[
            "elapsed_ns"
        ]

        if len(
            elapsed_ns
        ) != timed_runs:

            raise RuntimeError(
                f"Condition {execution_order} returned "
                "incorrect number of timed observations."
            )


        values_ms = (
            np.asarray(
                elapsed_ns,
                dtype=np.float64,
            )
            /
            1_000_000.0
        )


        p50_ms = float(
            np.percentile(
                values_ms,
                50,
            )
        )

        p95_ms = float(
            np.percentile(
                values_ms,
                95,
            )
        )

        throughput = (
            float(
                batch_size
            )
            /
            (
                np.asarray(
                    elapsed_ns,
                    dtype=np.float64,
                )
                /
                1e9
            )
        )


        median_throughput = float(
            np.percentile(
                throughput,
                50,
            )
        )


        print(
            f"[PASS] {execution_order:03d}/080 "
            f"{target:43s} "
            f"{hardware_mode:21s} "
            f"B={batch_size:5d} "
            f"p50={p50_ms:10.3f} ms "
            f"p95={p95_ms:10.3f} ms "
            f"median={median_throughput:12.1f} flow/s"
        )


    elif status == "RESOURCE_LIMIT_OOM":

        print(
            f"[OOM] {execution_order:03d}/080 "
            f"{target:43s} "
            f"{hardware_mode:21s} "
            f"B={batch_size:5d}"
        )


    elif status == "TIMEOUT_RESOURCE_LIMIT":

        print(
            f"[TIMEOUT] {execution_order:03d}/080 "
            f"{target:43s} "
            f"{hardware_mode:21s} "
            f"B={batch_size:5d}"
        )


    existing_final_orders.add(
        execution_order
    )

    new_condition_count += 1


# =============================================================================
# 12. ALL-80 CONDITION COMPLETENESS GATE
# =============================================================================

banner(
    "STAGE26-2 :: 80-CONDITION COMPLETENESS GATE"
)


receipt_paths = sorted(
    CONDITION_DIR.glob(
        "condition_*.json"
    )
)


if len(
    receipt_paths
) != 80:

    raise RuntimeError(
        f"Expected 80 final condition receipts; "
        f"found {len(receipt_paths)}."
    )


all_receipts = []


for receipt_path in receipt_paths:

    receipt = json.loads(
        receipt_path.read_text(
            encoding="utf-8"
        )
    )

    all_receipts.append(
        receipt
    )


all_receipts = sorted(
    all_receipts,
    key=lambda row: int(
        row[
            "execution_order"
        ]
    ),
)


orders = [
    int(
        row[
            "execution_order"
        ]
    )
    for row in all_receipts
]


if orders != list(
    range(
        1,
        81,
    )
):

    raise RuntimeError(
        "Condition receipt execution orders are not exactly 1..80."
    )


status_counts = {}


for receipt in all_receipts:

    status_counts[
        receipt[
            "status"
        ]
    ] = (
        status_counts.get(
            receipt[
                "status"
            ],
            0,
        )
        +
        1
    )


print(
    "Condition status counts:"
)

for status, count in sorted(
    status_counts.items()
):

    print(
        f"  {status:30s} {count}"
    )


for receipt in all_receipts:

    if receipt[
        "status"
    ] not in ALLOWED_FINAL_CONDITION_STATUSES:

        raise RuntimeError(
            "Non-final/non-permitted condition status remains."
        )


# =============================================================================
# 13. AGGREGATE ALL RAW TIMINGS
# =============================================================================

banner(
    "STAGE26-2 :: AGGREGATE RAW NANOSECOND TIMINGS"
)


raw_rows = []


for receipt in all_receipts:

    if receipt[
        "status"
    ] != "PASS":

        continue


    elapsed_ns = receipt[
        "elapsed_ns"
    ]

    batch_size = int(
        receipt[
            "batch_size"
        ]
    )


    for iteration_index, elapsed in enumerate(
        elapsed_ns,
        start=1,
    ):

        elapsed = int(
            elapsed
        )

        elapsed_seconds = (
            elapsed
            /
            1e9
        )

        flows_per_second = (
            float(
                batch_size
            )
            /
            elapsed_seconds
        )

        amortized_ns_per_flow = (
            float(
                elapsed
            )
            /
            float(
                batch_size
            )
        )


        raw_rows.append(
            {
                "execution_order":
                    int(
                        receipt[
                            "execution_order"
                        ]
                    ),

                "condition_id":
                    receipt[
                        "condition_id"
                    ],

                "target_id":
                    receipt[
                        "target_id"
                    ],

                "target_role":
                    receipt[
                        "target_role"
                    ],

                "comparison_group":
                    receipt[
                        "comparison_group"
                    ],

                "hardware_mode":
                    receipt[
                        "hardware_mode"
                    ],

                "thread_count":
                    int(
                        receipt[
                            "thread_count"
                        ]
                    ),

                "batch_size":
                    batch_size,

                "iteration_index":
                    iteration_index,

                "elapsed_ns":
                    elapsed,

                "amortized_ns_per_flow":
                    amortized_ns_per_flow,

                "flows_per_second":
                    flows_per_second,

                "status":
                    "PASS",
            }
        )


print(
    "Raw timed observations:",
    len(
        raw_rows
    ),
)


# Full-success expected count is 8,320.
# Resource-limited frozen conditions legitimately reduce this count.
FULL_SUCCESS_RAW_COUNT = 8320


print(
    "Full-success design count:",
    FULL_SUCCESS_RAW_COUNT,
)


# =============================================================================
# 14. WRITE MASTER RAW JSONL + CSV
# =============================================================================

with RAW_JSONL.open(
    "w",
    encoding="utf-8",
) as f:

    for row in raw_rows:

        f.write(
            json.dumps(
                row,
                sort_keys=True,
            )
            +
            "\n"
        )


raw_columns = [
    "execution_order",
    "condition_id",
    "target_id",
    "target_role",
    "comparison_group",
    "hardware_mode",
    "thread_count",
    "batch_size",
    "iteration_index",
    "elapsed_ns",
    "amortized_ns_per_flow",
    "flows_per_second",
    "status",
]


with RAW_CSV.open(
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=raw_columns,
    )

    writer.writeheader()

    writer.writerows(
        raw_rows
    )


# =============================================================================
# 15. CONDITION STATUS TABLE
# =============================================================================

condition_status_rows = []


for receipt in all_receipts:

    elapsed = np.asarray(
        receipt.get(
            "elapsed_ns",
            [],
        ),
        dtype=np.float64,
    )

    batch_size = int(
        receipt[
            "batch_size"
        ]
    )


    if (
        receipt[
            "status"
        ] == "PASS"
        and
        elapsed.size > 0
    ):

        elapsed_ms = (
            elapsed
            /
            1_000_000.0
        )

        throughput = (
            float(
                batch_size
            )
            /
            (
                elapsed
                /
                1e9
            )
        )


        p50_ms = float(
            np.percentile(
                elapsed_ms,
                50,
            )
        )

        p95_ms = float(
            np.percentile(
                elapsed_ms,
                95,
            )
        )

        p99_ms = (
            float(
                np.percentile(
                    elapsed_ms,
                    99,
                )
            )
            if elapsed.size >= 100
            else
            None
        )

        median_throughput = float(
            np.percentile(
                throughput,
                50,
            )
        )


    else:

        p50_ms = None

        p95_ms = None

        p99_ms = None

        median_throughput = None


    condition_status_rows.append(
        {
            "execution_order":
                int(
                    receipt[
                        "execution_order"
                    ]
                ),

            "condition_id":
                receipt[
                    "condition_id"
                ],

            "target_id":
                receipt[
                    "target_id"
                ],

            "target_role":
                receipt[
                    "target_role"
                ],

            "comparison_group":
                receipt[
                    "comparison_group"
                ],

            "hardware_mode":
                receipt[
                    "hardware_mode"
                ],

            "thread_count":
                int(
                    receipt[
                        "thread_count"
                    ]
                ),

            "batch_size":
                batch_size,

            "warmup_runs":
                int(
                    receipt[
                        "warmup_runs"
                    ]
                ),

            "timed_runs_planned":
                int(
                    receipt[
                        "timed_runs"
                    ]
                ),

            "timed_runs_observed":
                int(
                    elapsed.size
                ),

            "status":
                receipt[
                    "status"
                ],

            "p50_batch_latency_ms":
                p50_ms,

            "p95_batch_latency_ms":
                p95_ms,

            "p99_batch_latency_ms":
                p99_ms,

            "median_throughput_flows_per_second":
                median_throughput,
        }
    )


condition_status_columns = list(
    condition_status_rows[
        0
    ].keys()
)


with CONDITION_STATUS_CSV.open(
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=condition_status_columns,
    )

    writer.writeheader()

    writer.writerows(
        condition_status_rows
    )


# =============================================================================
# 16. STATISTICAL SUMMARY + FROZEN BOOTSTRAP CIs
# =============================================================================

banner(
    "STAGE26-2 :: STATISTICAL SUMMARY + BOOTSTRAP"
)


summary_rows = []


for receipt in all_receipts:

    if receipt[
        "status"
    ] != "PASS":

        continue


    elapsed_ns = np.asarray(
        receipt[
            "elapsed_ns"
        ],
        dtype=np.float64,
    )

    elapsed_ms = (
        elapsed_ns
        /
        1_000_000.0
    )

    batch_size = int(
        receipt[
            "batch_size"
        ]
    )

    amortized_ms = (
        elapsed_ms
        /
        float(
            batch_size
        )
    )

    throughput = (
        float(
            batch_size
        )
        /
        (
            elapsed_ns
            /
            1e9
        )
    )


    mean_ms = float(
        np.mean(
            elapsed_ms
        )
    )

    std_ms = float(
        np.std(
            elapsed_ms,
            ddof=1,
        )
    )

    cv = (
        float(
            std_ms
            /
            mean_ms
        )
        if mean_ms != 0
        else None
    )

    p50_ms = float(
        np.percentile(
            elapsed_ms,
            50,
        )
    )

    p95_ms = float(
        np.percentile(
            elapsed_ms,
            95,
        )
    )

    p99_ms = (
        float(
            np.percentile(
                elapsed_ms,
                99,
            )
        )
        if elapsed_ms.size >= 100
        else
        None
    )

    maximum_ms = float(
        np.max(
            elapsed_ms
        )
    )


    p50_ci = bootstrap_ci(
        elapsed_ms,
        lambda x: float(
            np.percentile(
                x,
                50,
            )
        ),
        seed=BOOTSTRAP_SEED,
    )


    p95_ci = bootstrap_ci(
        elapsed_ms,
        lambda x: float(
            np.percentile(
                x,
                95,
            )
        ),
        seed=BOOTSTRAP_SEED,
    )


    if elapsed_ms.size >= 100:

        p99_ci = bootstrap_ci(
            elapsed_ms,
            lambda x: float(
                np.percentile(
                    x,
                    99,
                )
            ),
            seed=BOOTSTRAP_SEED,
        )

    else:

        p99_ci = (
            None,
            None,
        )


    median_throughput = float(
        np.percentile(
            throughput,
            50,
        )
    )


    throughput_ci = bootstrap_ci(
        throughput,
        lambda x: float(
            np.percentile(
                x,
                50,
            )
        ),
        seed=BOOTSTRAP_SEED,
    )


    summary_rows.append(
        {
            "execution_order":
                int(
                    receipt[
                        "execution_order"
                    ]
                ),

            "condition_id":
                receipt[
                    "condition_id"
                ],

            "target_id":
                receipt[
                    "target_id"
                ],

            "target_role":
                receipt[
                    "target_role"
                ],

            "comparison_group":
                receipt[
                    "comparison_group"
                ],

            "hardware_mode":
                receipt[
                    "hardware_mode"
                ],

            "thread_count":
                int(
                    receipt[
                        "thread_count"
                    ]
                ),

            "batch_size":
                batch_size,

            "n":
                int(
                    elapsed_ms.size
                ),

            "mean_batch_latency_ms":
                mean_ms,

            "std_batch_latency_ms":
                std_ms,

            "coefficient_of_variation":
                cv,

            "p50_batch_latency_ms":
                p50_ms,

            "p95_batch_latency_ms":
                p95_ms,

            "p99_batch_latency_ms_if_n_gte_100":
                p99_ms,

            "maximum_batch_latency_ms_descriptive_only":
                maximum_ms,

            "p50_ci95_low_ms":
                p50_ci[
                    0
                ],

            "p50_ci95_high_ms":
                p50_ci[
                    1
                ],

            "p95_ci95_low_ms":
                p95_ci[
                    0
                ],

            "p95_ci95_high_ms":
                p95_ci[
                    1
                ],

            "p99_ci95_low_ms_if_n_gte_100":
                p99_ci[
                    0
                ],

            "p99_ci95_high_ms_if_n_gte_100":
                p99_ci[
                    1
                ],

            "median_amortized_latency_ms_per_flow":
                float(
                    np.percentile(
                        amortized_ms,
                        50,
                    )
                ),

            "median_throughput_flows_per_second":
                median_throughput,

            "median_throughput_ci95_low":
                throughput_ci[
                    0
                ],

            "median_throughput_ci95_high":
                throughput_ci[
                    1
                ],
        }
    )


summary_rows = sorted(
    summary_rows,
    key=lambda row: int(
        row[
            "execution_order"
        ]
    ),
)


if summary_rows:

    summary_columns = list(
        summary_rows[
            0
        ].keys()
    )


    with SUMMARY_CSV.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=summary_columns,
        )

        writer.writeheader()

        writer.writerows(
            summary_rows
        )


else:

    raise RuntimeError(
        "No PASS conditions available for Stage26-2 summary."
    )


# =============================================================================
# 17. AGGREGATED CONDITION RECEIPTS
# =============================================================================

write_json(
    CONDITION_RECEIPTS_JSON,
    {
        "schema":
            "stage26_2_condition_receipts_v1",

        "stage":
            26,

        "checkpoint":
            "STAGE26-2",

        "parent_commit":
            EXPECTED_HEAD,

        "condition_count":
            len(
                all_receipts
            ),

        "condition_receipts":
            all_receipts,
    },
)


# =============================================================================
# 18. SUMMARY JSON
# =============================================================================

summary_payload = {
    "schema":
        "stage26_2_warm_cpu_summary_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-2",

    "parent_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "cpu_execution_plan_sha256":
        EXPECTED_EXECUTION_PLAN_SHA256,

    "implementation_sha256":
        implementation_sha,

    "worker_sha256":
        worker_sha,

    "condition_count":
        80,

    "condition_status_counts":
        status_counts,

    "raw_timing_observation_count":
        len(
            raw_rows
        ),

    "full_success_raw_timing_design_count":
        FULL_SUCCESS_RAW_COUNT,

    "bootstrap_replicates":
        BOOTSTRAP_REPLICATES,

    "bootstrap_seed":
        BOOTSTRAP_SEED,

    "summary_rows":
        summary_rows,

    "claim_boundary":
        (
            "Stage26-2 establishes warm CPU model-inference latency and "
            "batch throughput under the frozen CPU conditions only. "
            "Memory, feature extraction, end-to-end pipeline, GPU, Pareto, "
            "capacity and bottleneck conclusions are not yet permitted."
        ),
}


write_json(
    SUMMARY_JSON,
    summary_payload,
)


# =============================================================================
# 19. DISPLAY PRIMARY BATCH-1 LATENCY TABLE
# =============================================================================

banner(
    "STAGE26-2 :: PRIMARY BATCH-1 CPU LATENCY SUMMARY"
)


batch1_rows = [
    row
    for row in summary_rows
    if int(
        row[
            "batch_size"
        ]
    ) == 1
]


for row in batch1_rows:

    p99_text = (
        f"{row['p99_batch_latency_ms_if_n_gte_100']:.4f}"
        if
        row[
            "p99_batch_latency_ms_if_n_gte_100"
        ] is not None
        else
        "NA"
    )


    print(
        f"{row['target_id']:43s} "
        f"{row['hardware_mode']:21s} "
        f"p50={row['p50_batch_latency_ms']:9.4f} ms "
        f"p95={row['p95_batch_latency_ms']:9.4f} ms "
        f"p99={p99_text:>9s} ms "
        f"median={row['median_throughput_flows_per_second']:11.1f} flow/s"
    )


# =============================================================================
# 20. DISPLAY BATCH-SCALING STATUS
# =============================================================================

banner(
    "STAGE26-2 :: CONDITION COMPLETION MATRIX"
)


for target in sorted(
    expected_targets
):

    print(
        "\n"
        + target
    )


    target_rows = [
        row
        for row in condition_status_rows
        if row[
            "target_id"
        ] == target
    ]


    target_rows = sorted(
        target_rows,
        key=lambda row: (
            row[
                "hardware_mode"
            ],
            int(
                row[
                    "batch_size"
                ]
            ),
        ),
    )


    for row in target_rows:

        if row[
            "status"
        ] == "PASS":

            print(
                f"  {row['hardware_mode']:21s} "
                f"B={int(row['batch_size']):5d} "
                f"PASS "
                f"p50={row['p50_batch_latency_ms']:10.3f} ms "
                f"median={row['median_throughput_flows_per_second']:12.1f} flow/s"
            )

        else:

            print(
                f"  {row['hardware_mode']:21s} "
                f"B={int(row['batch_size']):5d} "
                f"{row['status']}"
            )


# =============================================================================
# 21. BUILD STAGE26-2 RECEIPT
# =============================================================================

banner(
    "STAGE26-2 :: BUILD MEASUREMENT RECEIPT"
)


pass_conditions = int(
    status_counts.get(
        "PASS",
        0,
    )
)

oom_conditions = int(
    status_counts.get(
        "RESOURCE_LIMIT_OOM",
        0,
    )
)

timeout_conditions = int(
    status_counts.get(
        "TIMEOUT_RESOURCE_LIMIT",
        0,
    )
)


overall_status = (
    "PASS"
    if (
        pass_conditions == 80
    )
    else
    "COMPLETE_WITH_FROZEN_RESOURCE_LIMITS"
)


receipt = {
    "schema":
        "stage26_2_warm_cpu_receipt_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-2",

    "status":
        overall_status,

    "completed_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "cpu_execution_plan_sha256":
        EXPECTED_EXECUTION_PLAN_SHA256,

    "cpu_preflight_receipt_sha256":
        EXPECTED_PREFLIGHT_RECEIPT_SHA256,

    "cold_start_receipt_sha256":
        EXPECTED_COLD_RECEIPT_SHA256,

    "implementation_sha256":
        implementation_sha,

    "worker_sha256":
        worker_sha,

    "condition_count":
        80,

    "pass_condition_count":
        pass_conditions,

    "resource_limit_oom_condition_count":
        oom_conditions,

    "timeout_resource_limit_condition_count":
        timeout_conditions,

    "raw_timing_observation_count":
        len(
            raw_rows
        ),

    "full_success_raw_timing_design_count":
        FULL_SUCCESS_RAW_COUNT,

    "raw_jsonl_sha256":
        sha256_file(
            RAW_JSONL
        ),

    "raw_csv_sha256":
        sha256_file(
            RAW_CSV
        ),

    "condition_receipts_sha256":
        sha256_file(
            CONDITION_RECEIPTS_JSON
        ),

    "condition_status_csv_sha256":
        sha256_file(
            CONDITION_STATUS_CSV
        ),

    "summary_csv_sha256":
        sha256_file(
            SUMMARY_CSV
        ),

    "summary_json_sha256":
        sha256_file(
            SUMMARY_JSON
        ),

    "warm_cpu_inference_measured":
        True,

    "batch_throughput_measured":
        True,

    "memory_profiled":
        False,

    "feature_extraction_measured":
        False,

    "end_to_end_pipeline_measured":
        False,

    "gpu_used":
        False,

    "holdout_reopened":
        False,

    "pareto_analysis_performed":
        False,

    "capacity_analysis_performed":
        False,

    "claim_boundary":
        (
            "Warm CPU inference and batch throughput only. "
            "No memory, extraction, end-to-end, GPU, bottleneck, "
            "capacity, or Pareto conclusion is yet permitted."
        ),

    "next_action":
        (
            "GIT_ANCHOR_STAGE26_2_BEFORE_MEMORY_PROFILING"
        ),
}


write_json(
    RECEIPT_PATH,
    receipt,
)


receipt_sha = sha256_file(
    RECEIPT_PATH
)


print(
    "Stage26-2 receipt status:",
    overall_status,
)

print(
    "Receipt SHA256:"
)

print(
    " ",
    receipt_sha,
)


# =============================================================================
# 22. BUILD DURABLE REPOSITORY PACKAGE
# =============================================================================

banner(
    "STAGE26-2 :: BUILD DURABLE RESULTS PACKAGE"
)


REPO_STAGE26_2_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


files_to_copy = [
    WORKER_PATH,
    IMPLEMENTATION_PATH,
    CONDITION_RECEIPTS_JSON,
    CONDITION_STATUS_CSV,
    RAW_JSONL,
    RAW_CSV,
    SUMMARY_CSV,
    SUMMARY_JSON,
    RECEIPT_PATH,
]


for source in files_to_copy:

    destination = (
        REPO_STAGE26_2_DIR
        / source.name
    )

    shutil.copy2(
        source,
        destination,
    )

    print(
        "[COPIED]",
        source.name,
    )


PACKAGE_MANIFEST = (
    REPO_STAGE26_2_DIR
    / "stage26_2_warm_cpu_package_manifest.json"
)


package_files = []


for path in sorted(
    REPO_STAGE26_2_DIR.iterdir()
):

    if (
        path.is_file()
        and
        path.name
        !=
        PACKAGE_MANIFEST.name
    ):

        package_files.append(
            {
                "path":
                    str(
                        path.relative_to(
                            REPO_DIR
                        )
                    ),

                "size_bytes":
                    path.stat().st_size,

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )


package_manifest = {
    "schema":
        "stage26_2_warm_cpu_package_manifest_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-2",

    "status":
        "READY_FOR_GIT_ANCHOR",

    "parent_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "receipt_sha256":
        receipt_sha,

    "file_count_excluding_manifest":
        len(
            package_files
        ),

    "files":
        package_files,

    "scientific_boundary":
        {
            "warm_cpu_inference_measured":
                True,

            "batch_throughput_measured":
                True,

            "memory_profiled":
                False,

            "feature_extraction_measured":
                False,

            "end_to_end_pipeline_measured":
                False,

            "gpu_used":
                False,

            "holdout_reopened":
                False,

            "pareto_analysis_performed":
                False,
        },
}


write_json(
    PACKAGE_MANIFEST,
    package_manifest,
)


package_sha = sha256_file(
    PACKAGE_MANIFEST
)


print(
    "\nPackage manifest SHA256:"
)

print(
    " ",
    package_sha,
)


# =============================================================================
# 23. FINAL AUDIT
# =============================================================================

banner(
    "STAGE26-2 FINAL AUDIT"
)


status_after = git(
    "status",
    "--porcelain",
)


print(
    "Repository status:"
)

print(
    status_after
    if status_after
    else
    "<unexpected clean>"
)


if not status_after:

    raise RuntimeError(
        "Expected new Stage26-2 package."
    )


unexpected = []


for line in status_after.splitlines():

    path = line[
        3:
    ]

    if not path.startswith(
        "results/stage26_deployment_profiling/"
        "stage26_2_cpu_warm_inference/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository changes outside Stage26-2:\n"
        +
        "\n".join(
            unexpected
        )
    )


# Original upstream hashes must still remain unchanged.
for name, path, expected_sha in immutable_checks:

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:

        raise RuntimeError(
            f"Upstream immutable artifact changed during Stage26-2: {name}"
        )


# =============================================================================
# 24. CLOSURE
# =============================================================================

banner(
    "STAGE26-2 WARM CPU PROFILING COMPLETE"
)


print(
    "Parent commit:"
)

print(
    " ",
    EXPECTED_HEAD,
)


print(
    "\nFrozen CPU conditions:"
)

print(
    "  80 / 80 final condition receipts"
)


print(
    "\nCondition statuses:"
)

print(
    "  PASS                   :",
    pass_conditions,
)

print(
    "  RESOURCE_LIMIT_OOM     :",
    oom_conditions,
)

print(
    "  TIMEOUT_RESOURCE_LIMIT :",
    timeout_conditions,
)


print(
    "\nRaw timed observations:"
)

print(
    " ",
    len(
        raw_rows
    ),
)

print(
    "  full-success design count =",
    FULL_SUCCESS_RAW_COUNT,
)


print(
    "\nRAW JSONL SHA256:"
)

print(
    " ",
    sha256_file(
        RAW_JSONL
    ),
)


print(
    "\nRAW CSV SHA256:"
)

print(
    " ",
    sha256_file(
        RAW_CSV
    ),
)


print(
    "\nSUMMARY CSV SHA256:"
)

print(
    " ",
    sha256_file(
        SUMMARY_CSV
    ),
)


print(
    "\nRECEIPT SHA256:"
)

print(
    " ",
    receipt_sha,
)


print(
    "\nPACKAGE MANIFEST SHA256:"
)

print(
    " ",
    package_sha,
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  WARM CPU INFERENCE LATENCY MEASURED"
)

print(
    "  CPU BATCH THROUGHPUT MEASURED"
)

print(
    "  RAW PER-ITERATION NANOSECOND TIMINGS RETAINED"
)

print(
    "  FROZEN 80-CONDITION ORDER PRESERVED"
)

print(
    "  FROZEN OOM/TIMEOUT POLICY PRESERVED"
)

print(
    "  MEMORY NOT YET PROFILED"
)

print(
    "  FEATURE EXTRACTION NOT YET PROFILED"
)

print(
    "  END-TO-END PIPELINE NOT YET PROFILED"
)

print(
    "  PARETO ANALYSIS NOT YET PERFORMED"
)

print(
    "  HOLDOUT NOT REOPENED"
)

print(
    "  GPU NOT USED"
)


print(
    "\nCRITICAL NEXT ACTION:"
)

print(
    "  STAGE26-2-GIT — commit, push, and remotely verify all warm CPU"
)

print(
    "  raw timings BEFORE Stage26-3 memory/package profiling."
)


STAGE26-2 :: PARENT / PROTOCOL GATE
Expected HEAD : 46379b6d036008db4d60b056a66f4c01383e3298
Local HEAD    : 46379b6d036008db4d60b056a66f4c01383e3298
origin/main   : 46379b6d036008db4d60b056a66f4c01383e3298
Repository clean: True
measurement_protocol         PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
cpu_execution_plan           PASS b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363
cpu_preflight_receipt        PASS 4c7fc55a0e44e53587d385989dfec4f3f57f20768c7dd4a2d08e2936d2ba21e5
cold_start_receipt           PASS 35d988fbdcf7d84d54e593cd37f1204a005c2e854ae7128cc559629616d99c10

STAGE26-2 :: RESUME / DOUBLE-RUN GATE
Previously durable condition receipts: 0
[FRESH MODE] No Stage26-2 condition has yet been measured.

STAGE26-2 :: LOAD FROZEN CPU EXECUTION PLAN
Frozen conditions: 80
Frozen targets   : 8

First 15 execution orders:
001 ENS_LGBM_XGB_EQUAL                          CPU_1_PHYSICAL_CORE   B=    1 W= 50 T=200
002 STAGE20_MASKED_CNN_V1   

KeyboardInterrupt: 

In [17]:
# =============================================================================
# STAGE26-2-RECOVERY-A — SAFE INTERRUPTION / DURABLE-STATE AUDIT
#
# Run ONLY AFTER stopping the currently-running Stage26-2 cell once.
#
# PURPOSE
# -------
# - do NOT perform inference
# - do NOT perform timing
# - do NOT remeasure anything
# - identify exactly which Stage26-2 conditions are already durable
# - detect/terminate any orphan warm-worker left by the interrupted parent cell
# - verify all completed receipts against the frozen 80-condition plan
# - record the operator interruption transparently
#
# Conditions with existing final receipts WILL NOT be remeasured later.
# =============================================================================

from __future__ import annotations

import os
import json
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import psutil


# =============================================================================
# CONFIG
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26 = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

STEADY = (
    STAGE26
    / "steady_state"
)

CONDITION_DIR = (
    STEADY
    / "condition_receipts"
)

WORKER = (
    STAGE26
    / "workers"
    / "stage26_warm_cpu_worker.py"
)

IMPLEMENTATION = (
    STEADY
    / "stage26_2_warm_cpu_implementation.json"
)

RECOVERY_RECORD = (
    STEADY
    / "stage26_2_interruption_recovery_record.json"
)

PLAN = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_0_protocol_lock/"
      "cpu_execution_plan.json"
)


EXPECTED_HEAD = (
    "46379b6d036008db4d60b056a66f4c01383e3298"
)

EXPECTED_PLAN_SHA = (
    "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363"
)

EXPECTED_WORKER_SHA = (
    "2248f9c61b896bccf306167524b8be978993f49fc198327a6e280ee4553709b4"
)

EXPECTED_IMPLEMENTATION_SHA = (
    "678c53993799f66e1528808f46c9c27c64b8821dcb8de2ce83343f126f848d43"
)


FINAL_STATUSES = {
    "PASS",
    "RESOURCE_LIMIT_OOM",
    "TIMEOUT_RESOURCE_LIMIT",
}


# =============================================================================
# HELPERS
# =============================================================================

def banner(text):
    print("\n" + "=" * 110)
    print(text)
    print("=" * 110)


def git(*args):
    p = subprocess.run(
        ["git", *args],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def atomic_json(path, obj):
    tmp = Path(
        str(path) + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write("\n")
        f.flush()
        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


# =============================================================================
# 1. SCIENTIFIC / GIT GATE
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-A :: SCIENTIFIC GATE"
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print("Expected HEAD :", EXPECTED_HEAD)
print("Local HEAD    :", head)
print("origin/main   :", remote)
print("Repo clean    :", status == "")


if head != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected repository HEAD."
    )

if remote != EXPECTED_HEAD:
    raise RuntimeError(
        "origin/main changed during Stage26-2."
    )

if status:
    raise RuntimeError(
        "Repository unexpectedly dirty:\n"
        + status
    )


# =============================================================================
# 2. FROZEN IMPLEMENTATION GATE
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-A :: IMPLEMENTATION GATE"
)


for name, path, expected_sha in [
    (
        "CPU execution plan",
        PLAN,
        EXPECTED_PLAN_SHA,
    ),
    (
        "Stage26-2 worker",
        WORKER,
        EXPECTED_WORKER_SHA,
    ),
    (
        "Stage26-2 implementation",
        IMPLEMENTATION,
        EXPECTED_IMPLEMENTATION_SHA,
    ),
]:

    if not path.exists():
        raise FileNotFoundError(
            path
        )

    actual = sha256_file(
        path
    )

    ok = (
        actual
        ==
        expected_sha
    )

    print(
        f"{name:30s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual}"
    )

    if not ok:
        raise RuntimeError(
            f"Frozen Stage26-2 artifact changed: {name}"
        )


# =============================================================================
# 3. LOAD EXACT 80-CONDITION PLAN
# =============================================================================

plan = json.loads(
    PLAN.read_text(
        encoding="utf-8"
    )
)


conditions = sorted(
    plan["conditions"],
    key=lambda x: int(
        x["execution_order"]
    ),
)


if len(conditions) != 80:
    raise RuntimeError(
        "Expected exactly 80 conditions."
    )


condition_by_order = {
    int(row["execution_order"]):
        row
    for row in conditions
}


# =============================================================================
# 4. DETECT ORPHAN WORKER FROM INTERRUPTED CELL
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-A :: ORPHAN WORKER CHECK"
)


worker_string = str(
    WORKER
)


orphan_processes = []


for proc in psutil.process_iter(
    [
        "pid",
        "cmdline",
        "status",
    ]
):

    try:
        cmdline = (
            proc.info[
                "cmdline"
            ]
            or
            []
        )

        joined = " ".join(
            cmdline
        )

    except (
        psutil.NoSuchProcess,
        psutil.AccessDenied,
    ):
        continue


    if (
        worker_string
        in
        joined
        and
        proc.pid
        !=
        os.getpid()
    ):

        orphan_processes.append(
            proc
        )


print(
    "Warm-worker processes found:",
    len(
        orphan_processes
    ),
)


terminated_pids = []


for proc in orphan_processes:

    print(
        "  terminating PID",
        proc.pid,
    )

    try:
        proc.terminate()
        terminated_pids.append(
            proc.pid
        )

    except psutil.NoSuchProcess:
        pass


# No measurement is allowed to continue concurrently with recovery.
for proc in orphan_processes:

    try:
        proc.wait(
            timeout=3
        )

    except psutil.TimeoutExpired:

        print(
            "  force-killing PID",
            proc.pid,
        )

        try:
            proc.kill()
        except psutil.NoSuchProcess:
            pass


# Confirm none remains.
still_running = []


for proc in psutil.process_iter(
    [
        "pid",
        "cmdline",
    ]
):

    try:
        joined = " ".join(
            proc.info[
                "cmdline"
            ]
            or
            []
        )

    except (
        psutil.NoSuchProcess,
        psutil.AccessDenied,
    ):
        continue


    if (
        worker_string
        in
        joined
        and
        proc.pid
        !=
        os.getpid()
    ):

        still_running.append(
            proc.pid
        )


print(
    "Workers remaining:",
    still_running,
)


if still_running:
    raise RuntimeError(
        "A Stage26 warm worker is still active. "
        "Do not resume measurements."
    )


# =============================================================================
# 5. AUDIT ALL DURABLE CONDITION RECEIPTS
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-A :: DURABLE RECEIPT AUDIT"
)


receipt_paths = sorted(
    CONDITION_DIR.glob(
        "condition_*.json"
    )
)


receipts = []


for path in receipt_paths:

    receipt = json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )

    order = int(
        receipt[
            "execution_order"
        ]
    )


    if order not in condition_by_order:
        raise RuntimeError(
            f"Unknown receipt execution order: {order}"
        )


    frozen = condition_by_order[
        order
    ]


    for field in [
        "condition_id",
        "target_id",
        "hardware_mode",
        "thread_count",
        "batch_size",
        "warmup_runs",
        "timed_runs",
    ]:

        if receipt.get(
            field
        ) != frozen.get(
            field
        ):

            raise RuntimeError(
                f"Condition {order:03d} differs "
                f"from frozen plan at {field}."
            )


    if receipt[
        "status"
    ] not in FINAL_STATUSES:

        raise RuntimeError(
            f"Condition {order:03d} has non-final "
            f"status {receipt['status']!r}."
        )


    if receipt[
        "status"
    ] == "PASS":

        observed = len(
            receipt.get(
                "elapsed_ns",
                [],
            )
        )

        expected = int(
            frozen[
                "timed_runs"
            ]
        )

        if observed != expected:
            raise RuntimeError(
                f"Condition {order:03d}: "
                f"{observed} timings; expected {expected}."
            )


    receipts.append(
        receipt
    )


receipts = sorted(
    receipts,
    key=lambda x: int(
        x[
            "execution_order"
        ]
    ),
)


durable_orders = [
    int(
        r[
            "execution_order"
        ]
    )
    for r in receipts
]


print(
    "Durable final receipts:",
    len(
        receipts
    ),
)


for receipt in receipts:

    print(
        f"  {int(receipt['execution_order']):03d} "
        f"{receipt['status']:24s} "
        f"{receipt['target_id']:43s} "
        f"{receipt['hardware_mode']:21s} "
        f"B={int(receipt['batch_size']):5d}"
    )


# =============================================================================
# 6. CONTIGUITY GATE
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-A :: EXECUTION-ORDER CONTIGUITY"
)


if durable_orders:

    expected_prefix = list(
        range(
            1,
            max(
                durable_orders
            )
            + 1,
        )
    )

    if durable_orders != expected_prefix:

        raise RuntimeError(
            "Durable receipts do not form a contiguous "
            "prefix of the frozen execution order."
        )

    last_durable = max(
        durable_orders
    )

else:
    last_durable = 0


next_order = (
    last_durable
    +
    1
)


print(
    "Last durable execution order:",
    last_durable,
)


if next_order <= 80:

    next_condition = (
        condition_by_order[
            next_order
        ]
    )

    print(
        "Next unfinished order       :",
        next_order,
    )

    print(
        "Next target                 :",
        next_condition[
            "target_id"
        ],
    )

    print(
        "Next hardware               :",
        next_condition[
            "hardware_mode"
        ],
    )

    print(
        "Next batch                  :",
        next_condition[
            "batch_size"
        ],
    )

    print(
        "Next warmup runs            :",
        next_condition[
            "warmup_runs"
        ],
    )

    print(
        "Next timed runs             :",
        next_condition[
            "timed_runs"
        ],
    )

else:

    next_condition = None

    print(
        "All 80 conditions already have final receipts."
    )


# =============================================================================
# 7. STATUS COUNTS
# =============================================================================

status_counts = {}


for receipt in receipts:

    status_counts[
        receipt[
            "status"
        ]
    ] = (
        status_counts.get(
            receipt[
                "status"
            ],
            0,
        )
        +
        1
    )


print(
    "\nDurable status counts:"
)


for key, value in sorted(
    status_counts.items()
):

    print(
        f"  {key:28s}: {value}"
    )


# =============================================================================
# 8. FAILURE-MARKER AUDIT
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-A :: FAILURE MARKERS"
)


failure_markers = []


for name in [
    "stage26_2_invalid_environment.json",
    "stage26_2_worker_failure.json",
]:

    path = (
        STEADY
        / name
    )

    if path.exists():

        failure_markers.append(
            str(
                path
            )
        )

        print(
            "[FOUND]",
            path,
        )


if not failure_markers:

    print(
        "No methodological/environment failure marker exists."
    )


# =============================================================================
# 9. RECORD THE INTERRUPTION TRANSPARENTLY
# =============================================================================

recovery_record = {
    "schema":
        "stage26_2_interruption_recovery_record_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-2",

    "recorded_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "reason":
        (
            "Operator interrupted parent Stage26-2 notebook cell "
            "after apparent long-running condition. Recovery preserves "
            "all durable final condition receipts and does not remeasure them."
        ),

    "parent_commit":
        EXPECTED_HEAD,

    "worker_sha256":
        EXPECTED_WORKER_SHA,

    "implementation_sha256":
        EXPECTED_IMPLEMENTATION_SHA,

    "cpu_execution_plan_sha256":
        EXPECTED_PLAN_SHA,

    "terminated_orphan_worker_pids":
        terminated_pids,

    "durable_final_condition_count":
        len(
            receipts
        ),

    "durable_execution_orders":
        durable_orders,

    "last_durable_execution_order":
        last_durable,

    "next_unfinished_execution_order":
        (
            next_order
            if next_order <= 80
            else None
        ),

    "next_unfinished_condition":
        next_condition,

    "existing_final_receipts_will_be_remeasured":
        False,

    "warm_measurement_protocol_changed":
        False,

    "batch_policy_changed":
        False,

    "iteration_policy_changed":
        False,

    "timeout_policy_changed":
        False,

    "thread_policy_changed":
        False,

    "affinity_policy_changed":
        False,

    "gpu_used":
        False,

    "holdout_reopened":
        False,

    "failure_markers":
        failure_markers,
}


atomic_json(
    RECOVERY_RECORD,
    recovery_record,
)


print(
    "\nRecovery record:"
)

print(
    " ",
    RECOVERY_RECORD,
)

print(
    "SHA256:"
)

print(
    " ",
    sha256_file(
        RECOVERY_RECORD
    ),
)


# =============================================================================
# 10. CLOSURE
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-A COMPLETE"
)


print(
    "NO NEW INFERENCE PERFORMED"
)

print(
    "NO NEW TIMING PERFORMED"
)

print(
    "NO COMPLETED CONDITION REMEASURED"
)

print(
    "NO BATCH SIZE CHANGED"
)

print(
    "NO ITERATION COUNT CHANGED"
)

print(
    "NO TIMEOUT CHANGED"
)

print(
    "GPU NOT USED"
)

print(
    "HOLDOUT NOT REOPENED"
)


print(
    "\nDurable completed conditions:",
    len(
        receipts
    ),
    "/ 80",
)


if next_condition is not None:

    print(
        "\nNEXT UNFINISHED FROZEN CONDITION:"
    )

    print(
        f"  order    : {next_order}"
    )

    print(
        f"  target   : {next_condition['target_id']}"
    )

    print(
        f"  hardware : {next_condition['hardware_mode']}"
    )

    print(
        f"  batch    : {next_condition['batch_size']}"
    )

    print(
        f"  warmup   : {next_condition['warmup_runs']}"
    )

    print(
        f"  timed    : {next_condition['timed_runs']}"
    )


print(
    "\nSEND THIS COMPLETE OUTPUT BEFORE RESUMING STAGE26-2."
)


STAGE26-2-RECOVERY-A :: SCIENTIFIC GATE
Expected HEAD : 46379b6d036008db4d60b056a66f4c01383e3298
Local HEAD    : 46379b6d036008db4d60b056a66f4c01383e3298
origin/main   : 46379b6d036008db4d60b056a66f4c01383e3298
Repo clean    : True

STAGE26-2-RECOVERY-A :: IMPLEMENTATION GATE
CPU execution plan             PASS b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363
Stage26-2 worker               PASS 2248f9c61b896bccf306167524b8be978993f49fc198327a6e280ee4553709b4
Stage26-2 implementation       PASS 678c53993799f66e1528808f46c9c27c64b8821dcb8de2ce83343f126f848d43

STAGE26-2-RECOVERY-A :: ORPHAN WORKER CHECK
Warm-worker processes found: 0
Workers remaining: []

STAGE26-2-RECOVERY-A :: DURABLE RECEIPT AUDIT
Durable final receipts: 20
  001 PASS                     ENS_LGBM_XGB_EQUAL                          CPU_1_PHYSICAL_CORE   B=    1
  002 PASS                     STAGE20_MASKED_CNN_V1                       CPU_1_PHYSICAL_CORE   B=    1
  003 PASS                     STAGE1

In [18]:
# =============================================================================
# STAGE26-2-RECOVERY-B
# EXECUTE ONLY FROZEN CONDITION 021
#
# Frozen condition:
#   order    = 21
#   target   = FT_BALANCED_SINGLE_RESOURCE_REFERENCE
#   hardware = CPU_1_PHYSICAL_CORE
#   batch    = 8192
#   warmup   = 5
#   timed    = 20
#
# IMPORTANT
# ---------
# - conditions 001..020 are NOT remeasured
# - frozen worker is NOT modified
# - frozen batch/warmup/timed/thread/affinity/timeout are NOT modified
# - heartbeat is parent-side operational visibility only
# - child timing remains time.perf_counter_ns inside frozen worker
# - memory profiling remains OFF
# - GPU remains OFF
# - holdout remains closed
# =============================================================================

from __future__ import annotations

import os
import json
import time
import signal
import hashlib
import subprocess
from pathlib import Path

import psutil


# =============================================================================
# 0. PATHS / FROZEN IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26 = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

STEADY = (
    STAGE26
    / "steady_state"
)

CONDITION_DIR = (
    STEADY
    / "condition_receipts"
)

CONFIG_DIR = (
    STEADY
    / "condition_configs"
)

WORKER = (
    STAGE26
    / "workers"
    / "stage26_warm_cpu_worker.py"
)

IMPLEMENTATION = (
    STEADY
    / "stage26_2_warm_cpu_implementation.json"
)

RECOVERY_RECORD = (
    STEADY
    / "stage26_2_interruption_recovery_record.json"
)

PLAN = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_0_protocol_lock/"
      "cpu_execution_plan.json"
)


EXPECTED_HEAD = (
    "46379b6d036008db4d60b056a66f4c01383e3298"
)

EXPECTED_PLAN_SHA256 = (
    "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363"
)

EXPECTED_WORKER_SHA256 = (
    "2248f9c61b896bccf306167524b8be978993f49fc198327a6e280ee4553709b4"
)

EXPECTED_IMPLEMENTATION_SHA256 = (
    "678c53993799f66e1528808f46c9c27c64b8821dcb8de2ce83343f126f848d43"
)

EXPECTED_RECOVERY_SHA256 = (
    "55e95ccc3426f52379931ff1126e4a6d8dfbf06ca75cd1c90dda18c078c6dcf8"
)


EXECUTION_ORDER = 21

MEASUREMENT_SEED = 26042

CPU_UTILIZATION_GATE_PERCENT = 20.0

MIN_AVAILABLE_RAM_GIB = 8.0

ENVIRONMENT_RETRY_COUNT = 3

ENVIRONMENT_RETRY_COOLDOWN_SECONDS = 5

CPU_UTILIZATION_SAMPLE_SECONDS = 1.0

CONDITION_COOLDOWN_SECONDS = 2

CONDITION_TIMEOUT_SECONDS = 600

HEARTBEAT_SECONDS = 30


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):
    print(
        "\n"
        + "=" * 112
    )

    print(text)

    print(
        "=" * 112
    )


def git(*args):

    p = subprocess.run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


# =============================================================================
# 2. SCIENTIFIC STATE GATE
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-B :: SCIENTIFIC STATE GATE"
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

repo_status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD :",
    EXPECTED_HEAD,
)

print(
    "Local HEAD    :",
    head,
)

print(
    "origin/main   :",
    remote,
)

print(
    "Repo clean    :",
    repo_status == "",
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected repository HEAD."
    )


if remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed during Stage26-2."
    )


if repo_status:

    raise RuntimeError(
        "Repository unexpectedly dirty:\n"
        + repo_status
    )


# =============================================================================
# 3. EXACT FROZEN IMPLEMENTATION GATE
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-B :: FROZEN IMPLEMENTATION GATE"
)


checks = [
    (
        "CPU execution plan",
        PLAN,
        EXPECTED_PLAN_SHA256,
    ),
    (
        "warm CPU worker",
        WORKER,
        EXPECTED_WORKER_SHA256,
    ),
    (
        "warm implementation",
        IMPLEMENTATION,
        EXPECTED_IMPLEMENTATION_SHA256,
    ),
    (
        "interruption recovery record",
        RECOVERY_RECORD,
        EXPECTED_RECOVERY_SHA256,
    ),
]


for name, path, expected_sha in checks:

    if not path.exists():

        raise FileNotFoundError(
            path
        )


    actual_sha = sha256_file(
        path
    )

    ok = (
        actual_sha
        ==
        expected_sha
    )


    print(
        f"{name:32s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual_sha}"
    )


    if not ok:

        raise RuntimeError(
            f"Frozen artifact changed: {name}"
        )


# =============================================================================
# 4. LOAD EXACT CONDITION 021 FROM FROZEN PLAN
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-B :: FROZEN CONDITION 021"
)


plan = json.loads(
    PLAN.read_text(
        encoding="utf-8"
    )
)


conditions = {
    int(
        row[
            "execution_order"
        ]
    ):
        row
    for row in plan[
        "conditions"
    ]
}


condition = conditions[
    EXECUTION_ORDER
]


print(
    "condition_id :",
    condition[
        "condition_id"
    ],
)

print(
    "target       :",
    condition[
        "target_id"
    ],
)

print(
    "role         :",
    condition[
        "target_role"
    ],
)

print(
    "group        :",
    condition[
        "comparison_group"
    ],
)

print(
    "hardware     :",
    condition[
        "hardware_mode"
    ],
)

print(
    "threads      :",
    condition[
        "thread_count"
    ],
)

print(
    "affinity     :",
    condition[
        "affinity"
    ],
)

print(
    "batch        :",
    condition[
        "batch_size"
    ],
)

print(
    "warmup       :",
    condition[
        "warmup_runs"
    ],
)

print(
    "timed        :",
    condition[
        "timed_runs"
    ],
)


# Exact identity check against audited expected condition.
expected_identity = {
    "target_id":
        "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",

    "hardware_mode":
        "CPU_1_PHYSICAL_CORE",

    "thread_count":
        1,

    "affinity":
        [0],

    "batch_size":
        8192,

    "warmup_runs":
        5,

    "timed_runs":
        20,
}


for key, expected in expected_identity.items():

    actual = condition.get(
        key
    )

    if actual != expected:

        raise RuntimeError(
            f"Condition 021 mismatch at {key}: "
            f"{actual!r} != {expected!r}"
        )


print(
    "\n[PASS] Condition 021 exactly matches frozen plan."
)


# =============================================================================
# 5. VERIFY CONDITIONS 001..020 REMAIN DURABLE
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-B :: PRIOR RECEIPT GATE"
)


for order in range(
    1,
    21,
):

    path = (
        CONDITION_DIR
        / f"condition_{order:03d}.json"
    )


    if not path.exists():

        raise RuntimeError(
            f"Prior durable condition disappeared: {order:03d}"
        )


print(
    "[PASS] Conditions 001..020 remain durable."
)


result_path = (
    CONDITION_DIR
    / "condition_021.json"
)


if result_path.exists():

    existing = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )

    print(
        "\nCondition 021 already has a receipt:"
    )

    print(
        json.dumps(
            existing,
            indent=2,
            sort_keys=True,
        )
    )

    raise RuntimeError(
        "Condition 021 already completed after the previous audit. "
        "Do not remeasure it."
    )


# =============================================================================
# 6. NO FAILURE MARKERS
# =============================================================================

for marker_name in [
    "stage26_2_invalid_environment.json",
    "stage26_2_worker_failure.json",
]:

    marker = (
        STEADY
        / marker_name
    )

    if marker.exists():

        raise RuntimeError(
            f"Existing Stage26-2 failure marker: {marker}"
        )


# =============================================================================
# 7. FROZEN ENVIRONMENT GATE
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-B :: ENVIRONMENT GATE"
)


environment_attempts = []

environment_passed = False

environment_final = None


for attempt in range(
    1,
    ENVIRONMENT_RETRY_COUNT + 1,
):

    cpu_percent = float(
        psutil.cpu_percent(
            interval=CPU_UTILIZATION_SAMPLE_SECONDS
        )
    )

    available_ram_gib = float(
        psutil.virtual_memory().available
        /
        1024**3
    )


    passed = (
        cpu_percent
        <=
        CPU_UTILIZATION_GATE_PERCENT
        and
        available_ram_gib
        >=
        MIN_AVAILABLE_RAM_GIB
    )


    record = {
        "attempt":
            attempt,

        "cpu_percent":
            cpu_percent,

        "available_ram_gib":
            available_ram_gib,

        "passed":
            passed,
    }


    environment_attempts.append(
        record
    )

    environment_final = record


    print(
        f"attempt={attempt} "
        f"cpu={cpu_percent:.1f}% "
        f"available_ram={available_ram_gib:.3f} GiB "
        f"{'PASS' if passed else 'FAIL'}"
    )


    if passed:

        environment_passed = True

        break


    if attempt < ENVIRONMENT_RETRY_COUNT:

        time.sleep(
            ENVIRONMENT_RETRY_COOLDOWN_SECONDS
        )


if not environment_passed:

    marker = (
        STEADY
        / "stage26_2_invalid_environment.json"
    )


    atomic_json(
        marker,
        {
            "schema":
                "stage26_2_invalid_environment_v1",

            "status":
                "INVALID_ENVIRONMENT",

            "execution_order":
                EXECUTION_ORDER,

            "condition_id":
                condition[
                    "condition_id"
                ],

            "target_id":
                condition[
                    "target_id"
                ],

            "hardware_mode":
                condition[
                    "hardware_mode"
                ],

            "batch_size":
                condition[
                    "batch_size"
                ],

            "environment_attempts":
                environment_attempts,

            "completed_condition_orders":
                list(
                    range(
                        1,
                        21,
                    )
                ),
        },
    )


    raise RuntimeError(
        "INVALID_ENVIRONMENT under the frozen Stage26 rule."
    )


# Frozen condition cooldown.
time.sleep(
    CONDITION_COOLDOWN_SECONDS
)


# =============================================================================
# 8. BUILD EXACT FROZEN WORKER CONFIG
# =============================================================================

config = {
    "schema":
        "stage26_2_worker_config_v1",

    "repo":
        str(
            REPO
        ),

    "result_path":
        str(
            result_path
        ),

    "measurement_seed":
        MEASUREMENT_SEED,

    "condition_id":
        condition[
            "condition_id"
        ],

    "execution_order":
        EXECUTION_ORDER,

    "target_id":
        condition[
            "target_id"
        ],

    "target_role":
        condition[
            "target_role"
        ],

    "comparison_group":
        condition[
            "comparison_group"
        ],

    "hardware_mode":
        condition[
            "hardware_mode"
        ],

    "thread_count":
        int(
            condition[
                "thread_count"
            ]
        ),

    "affinity":
        [
            int(x)
            for x in condition[
                "affinity"
            ]
        ],

    "batch_size":
        int(
            condition[
                "batch_size"
            ]
        ),

    "warmup_runs":
        int(
            condition[
                "warmup_runs"
            ]
        ),

    "timed_runs":
        int(
            condition[
                "timed_runs"
            ]
        ),
}


config_path = (
    CONFIG_DIR
    / "condition_021.json"
)


atomic_json(
    config_path,
    config,
)


# =============================================================================
# 9. EXACT FROZEN THREAD / CPU ENVIRONMENT
# =============================================================================

env = os.environ.copy()


thread_value = str(
    condition[
        "thread_count"
    ]
)


for key in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "BLIS_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
]:

    env[
        key
    ] = thread_value


env[
    "CUDA_VISIBLE_DEVICES"
] = ""


# =============================================================================
# 10. EXECUTE CONDITION 021 WITH PARENT-SIDE HEARTBEAT
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-B :: EXECUTE CONDITION 021"
)


print(
    "Frozen worker unchanged."
)

print(
    "Timeout remains 600 seconds."
)

print(
    "Heartbeat every 30 seconds is parent-side visibility only."
)

print(
    "This condition may be substantially slower than previous conditions."
)


process = subprocess.Popen(
    [
        sys.executable,
        str(
            WORKER
        ),
        str(
            config_path
        ),
    ],
    cwd=REPO,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)


operational_start = (
    time.monotonic()
)

timed_out = False

heartbeat_number = 0


while True:

    returncode = process.poll()


    if returncode is not None:

        break


    operational_elapsed = (
        time.monotonic()
        -
        operational_start
    )


    if operational_elapsed >= CONDITION_TIMEOUT_SECONDS:

        timed_out = True

        print(
            "\n[TIMEOUT BOUNDARY] "
            "Frozen 600-second condition timeout reached."
        )

        process.kill()

        break


    # Sleep without consuming a benchmark CPU core.
    sleep_for = min(
        HEARTBEAT_SECONDS,
        CONDITION_TIMEOUT_SECONDS
        -
        operational_elapsed,
    )


    time.sleep(
        sleep_for
    )


    if process.poll() is None:

        heartbeat_number += 1

        elapsed_now = int(
            time.monotonic()
            -
            operational_start
        )

        print(
            f"[HEARTBEAT {heartbeat_number:02d}] "
            f"condition 021 still running; "
            f"operational wall time ≈ {elapsed_now}s / 600s timeout",
            flush=True,
        )


stdout, stderr = process.communicate()


# =============================================================================
# 11. FROZEN TIMEOUT / RESOURCE-LIMIT HANDLING
# =============================================================================

if timed_out:

    timeout_receipt = {
        "schema":
            "stage26_2_condition_receipt_v1",

        "status":
            "TIMEOUT_RESOURCE_LIMIT",

        "condition_id":
            condition[
                "condition_id"
            ],

        "execution_order":
            EXECUTION_ORDER,

        "target_id":
            condition[
                "target_id"
            ],

        "target_role":
            condition[
                "target_role"
            ],

        "comparison_group":
            condition[
                "comparison_group"
            ],

        "hardware_mode":
            condition[
                "hardware_mode"
            ],

        "thread_count":
            int(
                condition[
                    "thread_count"
                ]
            ),

        "affinity_requested":
            condition[
                "affinity"
            ],

        "batch_size":
            int(
                condition[
                    "batch_size"
                ]
            ),

        "warmup_runs":
            int(
                condition[
                    "warmup_runs"
                ]
            ),

        "timed_runs":
            int(
                condition[
                    "timed_runs"
                ]
            ),

        "timeout_seconds":
            CONDITION_TIMEOUT_SECONDS,

        "environment_attempts":
            environment_attempts,

        "environment_final":
            environment_final,

        "elapsed_ns":
            [],

        "memory_profiled":
            False,

        "holdout_accessed":
            False,

        "gpu_used":
            False,
    }


    atomic_json(
        result_path,
        timeout_receipt,
    )


# Worker died before writing a receipt.
elif not result_path.exists():

    if process.returncode in {
        -signal.SIGKILL,
        137,
    }:

        oom_receipt = {
            "schema":
                "stage26_2_condition_receipt_v1",

            "status":
                "RESOURCE_LIMIT_OOM",

            "resource_limit_reason":
                (
                    "worker terminated by SIGKILL without "
                    "parent timeout; retained as frozen "
                    "resource-limit/OOM condition"
                ),

            "condition_id":
                condition[
                    "condition_id"
                ],

            "execution_order":
                EXECUTION_ORDER,

            "target_id":
                condition[
                    "target_id"
                ],

            "target_role":
                condition[
                    "target_role"
                ],

            "comparison_group":
                condition[
                    "comparison_group"
                ],

            "hardware_mode":
                condition[
                    "hardware_mode"
                ],

            "thread_count":
                int(
                    condition[
                        "thread_count"
                    ]
                ),

            "affinity_requested":
                condition[
                    "affinity"
                ],

            "batch_size":
                int(
                    condition[
                        "batch_size"
                    ]
                ),

            "warmup_runs":
                int(
                    condition[
                        "warmup_runs"
                    ]
                ),

            "timed_runs":
                int(
                    condition[
                        "timed_runs"
                    ]
                ),

            "returncode":
                process.returncode,

            "environment_attempts":
                environment_attempts,

            "environment_final":
                environment_final,

            "elapsed_ns":
                [],

            "memory_profiled":
                False,

            "holdout_accessed":
                False,

            "gpu_used":
                False,
        }


        atomic_json(
            result_path,
            oom_receipt,
        )


    else:

        marker = (
            STEADY
            / "stage26_2_worker_failure.json"
        )


        atomic_json(
            marker,
            {
                "schema":
                    "stage26_2_worker_failure_v1",

                "status":
                    "WORKER_FAILURE",

                "condition_id":
                    condition[
                        "condition_id"
                    ],

                "execution_order":
                    EXECUTION_ORDER,

                "target_id":
                    condition[
                        "target_id"
                    ],

                "hardware_mode":
                    condition[
                        "hardware_mode"
                    ],

                "batch_size":
                    condition[
                        "batch_size"
                    ],

                "returncode":
                    process.returncode,

                "stdout":
                    stdout[
                        -8000:
                    ],

                "stderr":
                    stderr[
                        -16000:
                    ],

                "environment_attempts":
                    environment_attempts,
            },
        )


        print(
            "\nWorker stdout:"
        )

        print(
            stdout
        )


        print(
            "\nWorker stderr:"
        )

        print(
            stderr
        )


        raise RuntimeError(
            "Condition 021 exited without a durable receipt."
        )


# =============================================================================
# 12. LOAD / NORMALIZE DURABLE RECEIPT
# =============================================================================

receipt = json.loads(
    result_path.read_text(
        encoding="utf-8"
    )
)


# Parent metadata, same behavior as original Stage26-2 cell.
receipt[
    "environment_attempts"
] = environment_attempts

receipt[
    "environment_final"
] = environment_final

receipt[
    "target_role"
] = condition[
    "target_role"
]

receipt[
    "comparison_group"
] = condition[
    "comparison_group"
]


atomic_json(
    result_path,
    receipt,
)


# =============================================================================
# 13. SCIENTIFIC RECEIPT VALIDATION
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-B :: CONDITION 021 RESULT"
)


print(
    "status:",
    receipt[
        "status"
    ],
)


if receipt[
    "status"
] == "WORKER_FAILURE":

    marker = (
        STEADY
        / "stage26_2_worker_failure.json"
    )


    atomic_json(
        marker,
        {
            "schema":
                "stage26_2_worker_failure_v1",

            "status":
                "WORKER_FAILURE",

            "condition_receipt":
                receipt,
        },
    )


    print(
        json.dumps(
            receipt,
            indent=2,
            sort_keys=True,
        )
    )


    raise RuntimeError(
        "Condition 021 returned WORKER_FAILURE."
    )


if receipt[
    "status"
] not in {
    "PASS",
    "RESOURCE_LIMIT_OOM",
    "TIMEOUT_RESOURCE_LIMIT",
}:

    raise RuntimeError(
        "Unexpected final status: "
        + str(
            receipt[
                "status"
            ]
        )
    )


# Frozen plan identity.
for field in [
    "condition_id",
    "target_id",
    "hardware_mode",
    "thread_count",
    "batch_size",
    "warmup_runs",
    "timed_runs",
]:

    if receipt.get(
        field
    ) != condition.get(
        field
    ):

        raise RuntimeError(
            f"Receipt identity mismatch at {field}."
        )


if receipt[
    "status"
] == "PASS":

    elapsed_ns = receipt[
        "elapsed_ns"
    ]


    if len(
        elapsed_ns
    ) != int(
        condition[
            "timed_runs"
        ]
    ):

        raise RuntimeError(
            "PASS receipt does not contain exactly "
            "20 frozen timing observations."
        )


    import numpy as np


    elapsed_ms = (
        np.asarray(
            elapsed_ns,
            dtype=np.float64,
        )
        /
        1_000_000.0
    )


    throughput = (
        float(
            condition[
                "batch_size"
            ]
        )
        /
        (
            np.asarray(
                elapsed_ns,
                dtype=np.float64,
            )
            /
            1e9
        )
    )


    print(
        "timed observations:",
        len(
            elapsed_ns
        ),
    )

    print(
        "p50 batch latency :",
        f"{np.percentile(elapsed_ms, 50):.3f} ms",
    )

    print(
        "p95 batch latency :",
        f"{np.percentile(elapsed_ms, 95):.3f} ms",
    )

    print(
        "median throughput :",
        f"{np.percentile(throughput, 50):.1f} flow/s",
    )


elif receipt[
    "status"
] == "RESOURCE_LIMIT_OOM":

    print(
        "Frozen outcome: RESOURCE_LIMIT_OOM"
    )

    print(
        "Batch size remains 8192; no smaller replacement run is allowed."
    )


elif receipt[
    "status"
] == "TIMEOUT_RESOURCE_LIMIT":

    print(
        "Frozen outcome: TIMEOUT_RESOURCE_LIMIT"
    )

    print(
        "The 600-second timeout remains unchanged."
    )


# =============================================================================
# 14. FINAL DURABILITY AUDIT
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-B :: FINAL DURABILITY AUDIT"
)


durable_paths = sorted(
    CONDITION_DIR.glob(
        "condition_*.json"
    )
)


durable_orders = []


for path in durable_paths:

    row = json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )

    durable_orders.append(
        int(
            row[
                "execution_order"
        ]
    )
    )


durable_orders = sorted(
    durable_orders
)


print(
    "Durable receipt count:",
    len(
        durable_orders
    ),
)

print(
    "Durable orders:",
    durable_orders,
)


expected_prefix = list(
    range(
        1,
        22,
    )
)


if durable_orders != expected_prefix:

    raise RuntimeError(
        "After condition 021, durable receipts are not exactly 001..021."
    )


print(
    "\n[PASS] Conditions 001..021 now form a durable contiguous prefix."
)


print(
    "\nCondition 021 receipt:"
)

print(
    " ",
    result_path,
)

print(
    "SHA256:"
)

print(
    " ",
    sha256_file(
        result_path
    ),
)


# =============================================================================
# 15. CLOSURE
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-B COMPLETE"
)


print(
    "CONDITIONS 001..020 NOT REMEASURED"
)

print(
    "CONDITION 021 EXECUTED EXACTLY ONCE"
)

print(
    "FROZEN WORKER UNCHANGED"
)

print(
    "BATCH SIZE UNCHANGED"
)

print(
    "WARMUP COUNT UNCHANGED"
)

print(
    "TIMED COUNT UNCHANGED"
)

print(
    "THREAD POLICY UNCHANGED"
)

print(
    "CPU AFFINITY UNCHANGED"
)

print(
    "600-SECOND TIMEOUT UNCHANGED"
)

print(
    "MEMORY PROFILING NOT PERFORMED"
)

print(
    "GPU NOT USED"
)

print(
    "HOLDOUT NOT REOPENED"
)


print(
    "\nNEXT FROZEN EXECUTION ORDER: 022"
)

print(
    "Do not manually change or delete any Stage26 steady-state files."
)


STAGE26-2-RECOVERY-B :: SCIENTIFIC STATE GATE
Expected HEAD : 46379b6d036008db4d60b056a66f4c01383e3298
Local HEAD    : 46379b6d036008db4d60b056a66f4c01383e3298
origin/main   : 46379b6d036008db4d60b056a66f4c01383e3298
Repo clean    : True

STAGE26-2-RECOVERY-B :: FROZEN IMPLEMENTATION GATE
CPU execution plan               PASS b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363
warm CPU worker                  PASS 2248f9c61b896bccf306167524b8be978993f49fc198327a6e280ee4553709b4
warm implementation              PASS 678c53993799f66e1528808f46c9c27c64b8821dcb8de2ce83343f126f848d43
interruption recovery record     PASS 55e95ccc3426f52379931ff1126e4a6d8dfbf06ca75cd1c90dda18c078c6dcf8

STAGE26-2-RECOVERY-B :: FROZEN CONDITION 021
condition_id : CPUCOND_075
target       : FT_BALANCED_SINGLE_RESOURCE_REFERENCE
role         : RESOURCE_DECOMPOSITION_REFERENCE
group        : GROUP_A_DUPSAFE70
hardware     : CPU_1_PHYSICAL_CORE
threads      : 1
affinity     : [0]
batch        : 8192

In [19]:
# =============================================================================
# STAGE26-2-RECOVERY-C
# RESUME FROZEN WARM CPU PLAN FROM EXECUTION ORDER 022 THROUGH 080
#
# CONDITIONS 001..021:
#   AUDIT + SKIP ONLY. NEVER REMEASURE.
#
# CONDITIONS 022..080:
#   Execute exact frozen condition with exact frozen Stage26-2 worker.
#
# HEARTBEAT:
#   Parent-side operational visibility every 30 seconds.
#   It is OUTSIDE the measured T_infer boundary.
#
# NO SCIENTIFIC POLICY CHANGES:
#   - worker unchanged
#   - execution plan unchanged
#   - order unchanged
#   - batch sizes unchanged
#   - warmup counts unchanged
#   - timed counts unchanged
#   - thread counts unchanged
#   - affinities unchanged
#   - timeout unchanged (600 s)
#   - environment gate unchanged
#   - GPU OFF
#   - memory profiling OFF
#   - holdout CLOSED
# =============================================================================

from __future__ import annotations

import os
import sys
import csv
import json
import time
import signal
import shutil
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import psutil


# =============================================================================
# 0. FROZEN IDENTITIES / PATHS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26 = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

WORKERS_DIR = (
    STAGE26
    / "workers"
)

STEADY = (
    STAGE26
    / "steady_state"
)

CONDITION_DIR = (
    STEADY
    / "condition_receipts"
)

CONFIG_DIR = (
    STEADY
    / "condition_configs"
)

WORKER = (
    WORKERS_DIR
    / "stage26_warm_cpu_worker.py"
)

IMPLEMENTATION = (
    STEADY
    / "stage26_2_warm_cpu_implementation.json"
)

RECOVERY_A_RECORD = (
    STEADY
    / "stage26_2_interruption_recovery_record.json"
)

RESUME_RECORD = (
    STEADY
    / "stage26_2_resume_execution_record.json"
)

PLAN = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_0_protocol_lock/"
      "cpu_execution_plan.json"
)

PROTOCOL = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_0_protocol_lock/"
      "measurement_protocol.json"
)

PREFLIGHT_RECEIPT = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_0d_cpu_preflight/"
      "stage26_0d_cpu_preflight_receipt.json"
)

COLD_RECEIPT = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_1_cpu_cold_start/"
      "stage26_1_cold_start_receipt.json"
)


EXPECTED_HEAD = (
    "46379b6d036008db4d60b056a66f4c01383e3298"
)

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

EXPECTED_PLAN_SHA256 = (
    "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363"
)

EXPECTED_PREFLIGHT_SHA256 = (
    "4c7fc55a0e44e53587d385989dfec4f3f57f20768c7dd4a2d08e2936d2ba21e5"
)

EXPECTED_COLD_SHA256 = (
    "35d988fbdcf7d84d54e593cd37f1204a005c2e854ae7128cc559629616d99c10"
)

EXPECTED_WORKER_SHA256 = (
    "2248f9c61b896bccf306167524b8be978993f49fc198327a6e280ee4553709b4"
)

EXPECTED_IMPLEMENTATION_SHA256 = (
    "678c53993799f66e1528808f46c9c27c64b8821dcb8de2ce83343f126f848d43"
)

EXPECTED_RECOVERY_A_SHA256 = (
    "55e95ccc3426f52379931ff1126e4a6d8dfbf06ca75cd1c90dda18c078c6dcf8"
)

EXPECTED_CONDITION_021_SHA256 = (
    "47fdea995c8fe5574e904f598f57405bc0867072dd01a1deb5d09ccd836bf884"
)


MEASUREMENT_SEED = 26042

BOOTSTRAP_SEED = 26042

BOOTSTRAP_REPLICATES = 2000

CPU_UTILIZATION_GATE_PERCENT = 20.0

MIN_AVAILABLE_RAM_GIB = 8.0

ENVIRONMENT_RETRY_COUNT = 3

ENVIRONMENT_RETRY_COOLDOWN_SECONDS = 5

CPU_UTILIZATION_SAMPLE_SECONDS = 1.0

CONDITION_COOLDOWN_SECONDS = 2

CONDITION_TIMEOUT_SECONDS = 600

HEARTBEAT_SECONDS = 30


FINAL_STATUSES = {
    "PASS",
    "RESOURCE_LIMIT_OOM",
    "TIMEOUT_RESOURCE_LIMIT",
}


# =============================================================================
# 1. FINAL AGGREGATE OUTPUTS
# =============================================================================

CONDITION_RECEIPTS_JSON = (
    STEADY
    / "stage26_2_condition_receipts.json"
)

CONDITION_STATUS_CSV = (
    STEADY
    / "stage26_2_condition_status.csv"
)

RAW_JSONL = (
    STEADY
    / "stage26_2_warm_raw.jsonl"
)

RAW_CSV = (
    STEADY
    / "stage26_2_warm_raw.csv"
)

SUMMARY_CSV = (
    STEADY
    / "stage26_2_warm_summary.csv"
)

SUMMARY_JSON = (
    STEADY
    / "stage26_2_warm_summary.json"
)

RECEIPT_PATH = (
    STEADY
    / "stage26_2_warm_cpu_receipt.json"
)

REPO_STAGE26_2 = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_2_cpu_warm_inference"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def banner(text):
    print(
        "\n"
        + "=" * 116
    )

    print(text)

    print(
        "=" * 116
    )


def git(*args):
    p = subprocess.run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json(
    path,
    obj,
):
    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def stable_seed(text):
    digest = hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).digest()

    return int.from_bytes(
        digest[:8],
        "little",
    ) % (
        2**32
    )


def bootstrap_ci(
    values,
    statistic,
    *,
    seed,
    replicates=BOOTSTRAP_REPLICATES,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    if values.size == 0:
        return (
            None,
            None,
        )

    rng = np.random.default_rng(
        int(
            seed
        )
    )

    n = int(
        values.size
    )

    estimates = np.empty(
        replicates,
        dtype=np.float64,
    )

    for i in range(
        replicates
    ):

        sample = values[
            rng.integers(
                0,
                n,
                size=n,
            )
        ]

        estimates[i] = statistic(
            sample
        )

    return (
        float(
            np.percentile(
                estimates,
                2.5,
            )
        ),
        float(
            np.percentile(
                estimates,
                97.5,
            )
        ),
    )


# =============================================================================
# 3. GIT + IMMUTABILITY GATE
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-C :: SCIENTIFIC / IMMUTABILITY GATE"
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

repo_status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD :",
    EXPECTED_HEAD,
)

print(
    "Local HEAD    :",
    head,
)

print(
    "origin/main   :",
    remote,
)

print(
    "Repository clean:",
    repo_status == "",
)


if head != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected local HEAD."
    )

if remote != EXPECTED_HEAD:
    raise RuntimeError(
        "origin/main changed during Stage26-2."
    )

if repo_status:
    raise RuntimeError(
        "Repository unexpectedly dirty:\n"
        + repo_status
    )


immutable_checks = [
    (
        "measurement protocol",
        PROTOCOL,
        EXPECTED_PROTOCOL_SHA256,
    ),
    (
        "CPU execution plan",
        PLAN,
        EXPECTED_PLAN_SHA256,
    ),
    (
        "CPU preflight receipt",
        PREFLIGHT_RECEIPT,
        EXPECTED_PREFLIGHT_SHA256,
    ),
    (
        "cold-start receipt",
        COLD_RECEIPT,
        EXPECTED_COLD_SHA256,
    ),
    (
        "warm CPU worker",
        WORKER,
        EXPECTED_WORKER_SHA256,
    ),
    (
        "warm implementation",
        IMPLEMENTATION,
        EXPECTED_IMPLEMENTATION_SHA256,
    ),
    (
        "interruption recovery record",
        RECOVERY_A_RECORD,
        EXPECTED_RECOVERY_A_SHA256,
    ),
    (
        "condition 021 receipt",
        CONDITION_DIR
        / "condition_021.json",
        EXPECTED_CONDITION_021_SHA256,
    ),
]


for name, path, expected_sha in immutable_checks:

    if not path.exists():
        raise FileNotFoundError(
            path
        )

    actual_sha = sha256_file(
        path
    )

    ok = (
        actual_sha
        ==
        expected_sha
    )

    print(
        f"{name:34s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual_sha}"
    )

    if not ok:
        raise RuntimeError(
            f"Frozen artifact changed: {name}"
        )


# =============================================================================
# 4. LOAD FROZEN 80-CONDITION PLAN
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-C :: FROZEN PLAN GATE"
)


plan = json.loads(
    PLAN.read_text(
        encoding="utf-8"
    )
)


conditions = sorted(
    plan[
        "conditions"
    ],
    key=lambda row: int(
        row[
            "execution_order"
        ]
    ),
)


if len(
    conditions
) != 80:
    raise RuntimeError(
        "Expected 80 frozen CPU conditions."
    )


condition_by_order = {
    int(
        row[
            "execution_order"
        ]
    ):
        row
    for row in conditions
}


# =============================================================================
# 5. AUDIT DURABLE PREFIX 001..021
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-C :: DURABLE PREFIX AUDIT"
)


durable_orders = []


for receipt_path in sorted(
    CONDITION_DIR.glob(
        "condition_*.json"
    )
):

    receipt = json.loads(
        receipt_path.read_text(
            encoding="utf-8"
        )
    )

    order = int(
        receipt[
            "execution_order"
        ]
    )

    frozen = condition_by_order.get(
        order
    )

    if frozen is None:
        raise RuntimeError(
            f"Unknown condition receipt: {order}"
        )


    for field in [
        "condition_id",
        "target_id",
        "hardware_mode",
        "thread_count",
        "batch_size",
        "warmup_runs",
        "timed_runs",
    ]:

        if receipt.get(
            field
        ) != frozen.get(
            field
        ):

            raise RuntimeError(
                f"Condition {order:03d} frozen identity mismatch "
                f"at {field}."
            )


    if receipt[
        "status"
    ] not in FINAL_STATUSES:

        raise RuntimeError(
            f"Condition {order:03d} has invalid status "
            f"{receipt['status']}."
        )


    if receipt[
        "status"
    ] == "PASS":

        if len(
            receipt.get(
                "elapsed_ns",
                [],
            )
        ) != int(
            frozen[
                "timed_runs"
            ]
        ):

            raise RuntimeError(
                f"Condition {order:03d} raw timing count mismatch."
            )


    durable_orders.append(
        order
    )


durable_orders = sorted(
    durable_orders
)


print(
    "Durable receipts:",
    len(
        durable_orders
    ),
)

print(
    "Last durable order:",
    max(
        durable_orders
    )
    if durable_orders
    else None,
)


if durable_orders != list(
    range(
        1,
        22,
    )
):
    raise RuntimeError(
        "Expected durable prefix exactly 001..021."
    )


print(
    "[PASS] Conditions 001..021 are intact and will be skipped."
)


# =============================================================================
# 6. FAILURE-MARKER GATE
# =============================================================================

for marker_name in [
    "stage26_2_invalid_environment.json",
    "stage26_2_worker_failure.json",
]:

    marker = (
        STEADY
        / marker_name
    )

    if marker.exists():

        raise RuntimeError(
            f"Existing non-resource failure marker: {marker}"
        )


# =============================================================================
# 7. ENVIRONMENT GATE
# =============================================================================

def environment_gate():

    attempts = []

    for attempt in range(
        1,
        ENVIRONMENT_RETRY_COUNT + 1,
    ):

        cpu_percent = float(
            psutil.cpu_percent(
                interval=CPU_UTILIZATION_SAMPLE_SECONDS
            )
        )

        available_ram_gib = float(
            psutil.virtual_memory().available
            /
            1024**3
        )

        passed = (
            cpu_percent
            <=
            CPU_UTILIZATION_GATE_PERCENT
            and
            available_ram_gib
            >=
            MIN_AVAILABLE_RAM_GIB
        )

        record = {
            "attempt":
                attempt,

            "cpu_percent":
                cpu_percent,

            "available_ram_gib":
                available_ram_gib,

            "passed":
                passed,
        }

        attempts.append(
            record
        )

        if passed:

            return (
                True,
                attempts,
                record,
            )


        if attempt < ENVIRONMENT_RETRY_COUNT:

            time.sleep(
                ENVIRONMENT_RETRY_COOLDOWN_SECONDS
            )


    return (
        False,
        attempts,
        attempts[-1],
    )


# =============================================================================
# 8. RUN ONE FROZEN CONDITION WITH HEARTBEAT
# =============================================================================

def run_condition(
    condition,
):

    order = int(
        condition[
            "execution_order"
        ]
    )

    target = condition[
        "target_id"
    ]

    mode = condition[
        "hardware_mode"
    ]

    batch_size = int(
        condition[
            "batch_size"
        ]
    )

    threads = int(
        condition[
            "thread_count"
        ]
    )

    affinity = [
        int(x)
        for x in condition[
            "affinity"
        ]
    ]

    warmup_runs = int(
        condition[
            "warmup_runs"
        ]
    )

    timed_runs = int(
        condition[
            "timed_runs"
        ]
    )


    result_path = (
        CONDITION_DIR
        / f"condition_{order:03d}.json"
    )


    if result_path.exists():

        raise RuntimeError(
            f"Condition {order:03d} already has a receipt."
        )


    # Frozen condition cooldown.
    time.sleep(
        CONDITION_COOLDOWN_SECONDS
    )


    env_ok, env_attempts, env_final = (
        environment_gate()
    )


    if not env_ok:

        marker = (
            STEADY
            / "stage26_2_invalid_environment.json"
        )


        atomic_json(
            marker,
            {
                "schema":
                    "stage26_2_invalid_environment_v1",

                "status":
                    "INVALID_ENVIRONMENT",

                "execution_order":
                    order,

                "condition_id":
                    condition[
                        "condition_id"
                    ],

                "target_id":
                    target,

                "hardware_mode":
                    mode,

                "batch_size":
                    batch_size,

                "environment_attempts":
                    env_attempts,

                "completed_condition_orders":
                    sorted(
                        [
                            int(
                                json.loads(
                                    path.read_text(
                                        encoding="utf-8"
                                    )
                                )[
                                    "execution_order"
                                ]
                            )
                            for path in CONDITION_DIR.glob(
                                "condition_*.json"
                            )
                        ]
                    ),
            },
        )


        raise RuntimeError(
            "INVALID_ENVIRONMENT under frozen Stage26 policy."
        )


    config = {
        "schema":
            "stage26_2_worker_config_v1",

        "repo":
            str(
                REPO
            ),

        "result_path":
            str(
                result_path
            ),

        "measurement_seed":
            MEASUREMENT_SEED,

        "condition_id":
            condition[
                "condition_id"
            ],

        "execution_order":
            order,

        "target_id":
            target,

        "target_role":
            condition[
                "target_role"
            ],

        "comparison_group":
            condition[
                "comparison_group"
            ],

        "hardware_mode":
            mode,

        "thread_count":
            threads,

        "affinity":
            affinity,

        "batch_size":
            batch_size,

        "warmup_runs":
            warmup_runs,

        "timed_runs":
            timed_runs,
    }


    config_path = (
        CONFIG_DIR
        / f"condition_{order:03d}.json"
    )


    atomic_json(
        config_path,
        config,
    )


    env = os.environ.copy()

    thread_value = str(
        threads
    )


    for key in [
        "OMP_NUM_THREADS",
        "MKL_NUM_THREADS",
        "OPENBLAS_NUM_THREADS",
        "NUMEXPR_NUM_THREADS",
        "BLIS_NUM_THREADS",
        "VECLIB_MAXIMUM_THREADS",
    ]:

        env[
            key
        ] = thread_value


    env[
        "CUDA_VISIBLE_DEVICES"
    ] = ""


    print(
        f"\n[START] {order:03d}/080 "
        f"{target:43s} "
        f"{mode:21s} "
        f"B={batch_size:5d} "
        f"W={warmup_runs:3d} "
        f"T={timed_runs:3d}",
        flush=True,
    )


    process = subprocess.Popen(
        [
            sys.executable,
            str(
                WORKER
            ),
            str(
                config_path
            ),
        ],
        cwd=REPO,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )


    operational_start = (
        time.monotonic()
    )

    heartbeat_index = 0

    timed_out = False


    while True:

        returncode = (
            process.poll()
        )


        if returncode is not None:
            break


        elapsed_wall = (
            time.monotonic()
            -
            operational_start
        )


        if elapsed_wall >= CONDITION_TIMEOUT_SECONDS:

            timed_out = True

            print(
                f"[TIMEOUT BOUNDARY] {order:03d} "
                f"reached frozen 600-second limit.",
                flush=True,
            )

            process.kill()

            break


        sleep_for = min(
            HEARTBEAT_SECONDS,
            CONDITION_TIMEOUT_SECONDS
            -
            elapsed_wall,
        )


        time.sleep(
            sleep_for
        )


        if process.poll() is None:

            heartbeat_index += 1

            current_wall = int(
                time.monotonic()
                -
                operational_start
            )

            print(
                f"[HEARTBEAT {order:03d}.{heartbeat_index:02d}] "
                f"still running; operational wall ≈ "
                f"{current_wall}s / 600s",
                flush=True,
            )


    stdout, stderr = (
        process.communicate()
    )


    # -------------------------------------------------------------------------
    # Frozen timeout outcome
    # -------------------------------------------------------------------------

    if timed_out:

        atomic_json(
            result_path,
            {
                "schema":
                    "stage26_2_condition_receipt_v1",

                "status":
                    "TIMEOUT_RESOURCE_LIMIT",

                "condition_id":
                    condition[
                        "condition_id"
                    ],

                "execution_order":
                    order,

                "target_id":
                    target,

                "target_role":
                    condition[
                        "target_role"
                    ],

                "comparison_group":
                    condition[
                        "comparison_group"
                    ],

                "hardware_mode":
                    mode,

                "thread_count":
                    threads,

                "affinity_requested":
                    affinity,

                "batch_size":
                    batch_size,

                "warmup_runs":
                    warmup_runs,

                "timed_runs":
                    timed_runs,

                "timeout_seconds":
                    CONDITION_TIMEOUT_SECONDS,

                "environment_attempts":
                    env_attempts,

                "environment_final":
                    env_final,

                "elapsed_ns":
                    [],

                "memory_profiled":
                    False,

                "holdout_accessed":
                    False,

                "gpu_used":
                    False,
            },
        )


    # -------------------------------------------------------------------------
    # No receipt: SIGKILL => frozen resource-limit category
    # -------------------------------------------------------------------------

    elif not result_path.exists():

        if process.returncode in {
            -signal.SIGKILL,
            137,
        }:

            atomic_json(
                result_path,
                {
                    "schema":
                        "stage26_2_condition_receipt_v1",

                    "status":
                        "RESOURCE_LIMIT_OOM",

                    "resource_limit_reason":
                        (
                            "worker terminated by SIGKILL without parent "
                            "timeout; retained as resource-limit/OOM"
                        ),

                    "condition_id":
                        condition[
                            "condition_id"
                        ],

                    "execution_order":
                        order,

                    "target_id":
                        target,

                    "target_role":
                        condition[
                            "target_role"
                        ],

                    "comparison_group":
                        condition[
                            "comparison_group"
                        ],

                    "hardware_mode":
                        mode,

                    "thread_count":
                        threads,

                    "affinity_requested":
                        affinity,

                    "batch_size":
                        batch_size,

                    "warmup_runs":
                        warmup_runs,

                    "timed_runs":
                        timed_runs,

                    "returncode":
                        process.returncode,

                    "environment_attempts":
                        env_attempts,

                    "environment_final":
                        env_final,

                    "elapsed_ns":
                        [],

                    "memory_profiled":
                        False,

                    "holdout_accessed":
                        False,

                    "gpu_used":
                        False,
                },
            )


        else:

            marker = (
                STEADY
                / "stage26_2_worker_failure.json"
            )


            atomic_json(
                marker,
                {
                    "schema":
                        "stage26_2_worker_failure_v1",

                    "status":
                        "WORKER_FAILURE",

                    "execution_order":
                        order,

                    "condition_id":
                        condition[
                            "condition_id"
                        ],

                    "target_id":
                        target,

                    "hardware_mode":
                        mode,

                    "batch_size":
                        batch_size,

                    "returncode":
                        process.returncode,

                    "stdout":
                        stdout[
                            -8000:
                        ],

                    "stderr":
                        stderr[
                            -16000:
                        ],

                    "environment_attempts":
                        env_attempts,
                },
            )


            print(
                "\nSTDOUT:"
            )

            print(
                stdout
            )

            print(
                "\nSTDERR:"
            )

            print(
                stderr
            )


            raise RuntimeError(
                f"Condition {order:03d} failed without durable "
                "resource-limit receipt."
            )


    # -------------------------------------------------------------------------
    # Normalize parent metadata
    # -------------------------------------------------------------------------

    receipt = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


    receipt[
        "environment_attempts"
    ] = env_attempts

    receipt[
        "environment_final"
    ] = env_final

    receipt[
        "target_role"
    ] = condition[
        "target_role"
    ]

    receipt[
        "comparison_group"
    ] = condition[
        "comparison_group"
    ]


    atomic_json(
        result_path,
        receipt,
    )


    # -------------------------------------------------------------------------
    # Identity / status validation
    # -------------------------------------------------------------------------

    for field in [
        "condition_id",
        "target_id",
        "hardware_mode",
        "thread_count",
        "batch_size",
        "warmup_runs",
        "timed_runs",
    ]:

        if receipt.get(
            field
        ) != condition.get(
            field
        ):

            raise RuntimeError(
                f"Condition {order:03d} receipt differs from frozen "
                f"plan at {field}."
            )


    status = receipt[
        "status"
    ]


    if status == "WORKER_FAILURE":

        marker = (
            STEADY
            / "stage26_2_worker_failure.json"
        )


        atomic_json(
            marker,
            {
                "schema":
                    "stage26_2_worker_failure_v1",

                "status":
                    "WORKER_FAILURE",

                "condition_receipt":
                    receipt,
            },
        )


        raise RuntimeError(
            f"Condition {order:03d} returned WORKER_FAILURE."
        )


    if status not in FINAL_STATUSES:

        raise RuntimeError(
            f"Unexpected condition status: {status}"
        )


    if status == "PASS":

        elapsed = receipt[
            "elapsed_ns"
        ]


        if len(
            elapsed
        ) != timed_runs:

            raise RuntimeError(
                f"Condition {order:03d} timing count mismatch."
            )


        values_ms = (
            np.asarray(
                elapsed,
                dtype=np.float64,
            )
            /
            1_000_000.0
        )


        throughput = (
            float(
                batch_size
            )
            /
            (
                np.asarray(
                    elapsed,
                    dtype=np.float64,
                )
                /
                1e9
            )
        )


        print(
            f"[PASS] {order:03d}/080 "
            f"p50={np.percentile(values_ms,50):10.3f} ms "
            f"p95={np.percentile(values_ms,95):10.3f} ms "
            f"median={np.percentile(throughput,50):12.1f} flow/s",
            flush=True,
        )


    elif status == "RESOURCE_LIMIT_OOM":

        print(
            f"[OOM] {order:03d}/080 "
            f"{target} B={batch_size}",
            flush=True,
        )


    elif status == "TIMEOUT_RESOURCE_LIMIT":

        print(
            f"[TIMEOUT] {order:03d}/080 "
            f"{target} B={batch_size}",
            flush=True,
        )


    return receipt


# =============================================================================
# 9. EXECUTE ONLY 022..080
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-C :: RESUME EXECUTION 022 → 080"
)


for order in range(
    22,
    81,
):

    result_path = (
        CONDITION_DIR
        / f"condition_{order:03d}.json"
    )


    # Safe resume if notebook interrupts again.
    if result_path.exists():

        receipt = json.loads(
            result_path.read_text(
                encoding="utf-8"
            )
        )


        if receipt.get(
            "status"
        ) not in FINAL_STATUSES:

            raise RuntimeError(
                f"Existing condition {order:03d} is not final."
            )


        frozen = condition_by_order[
            order
        ]


        for field in [
            "condition_id",
            "target_id",
            "hardware_mode",
            "thread_count",
            "batch_size",
            "warmup_runs",
            "timed_runs",
        ]:

            if receipt.get(
                field
            ) != frozen.get(
                field
            ):

                raise RuntimeError(
                    f"Existing receipt {order:03d} differs "
                    f"from frozen plan."
                )


        if (
            receipt[
                "status"
            ] == "PASS"
            and
            len(
                receipt.get(
                    "elapsed_ns",
                    [],
                )
            )
            !=
            int(
                frozen[
                    "timed_runs"
                ]
            )
        ):

            raise RuntimeError(
                f"Existing PASS receipt {order:03d} is incomplete."
            )


        print(
            f"[SKIP] {order:03d}/080 already durable "
            f"({receipt['status']})"
        )

        continue


    run_condition(
        condition_by_order[
            order
        ]
    )


# =============================================================================
# 10. ALL-80 COMPLETENESS AUDIT
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-C :: ALL-80 COMPLETENESS AUDIT"
)


all_receipts = []


for order in range(
    1,
    81,
):

    path = (
        CONDITION_DIR
        / f"condition_{order:03d}.json"
    )


    if not path.exists():

        raise RuntimeError(
            f"Missing final condition receipt {order:03d}."
        )


    receipt = json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


    frozen = condition_by_order[
        order
    ]


    for field in [
        "condition_id",
        "target_id",
        "hardware_mode",
        "thread_count",
        "batch_size",
        "warmup_runs",
        "timed_runs",
    ]:

        if receipt.get(
            field
        ) != frozen.get(
            field
        ):

            raise RuntimeError(
                f"Final condition {order:03d} identity mismatch."
            )


    if receipt[
        "status"
    ] not in FINAL_STATUSES:

        raise RuntimeError(
            f"Condition {order:03d} has invalid final status."
        )


    if receipt[
        "status"
    ] == "PASS":

        if len(
            receipt.get(
                "elapsed_ns",
                [],
            )
        ) != int(
            frozen[
                "timed_runs"
            ]
        ):

            raise RuntimeError(
                f"Condition {order:03d} has incomplete raw timings."
            )


    all_receipts.append(
        receipt
    )


status_counts = {}


for receipt in all_receipts:

    status_counts[
        receipt[
            "status"
        ]
    ] = (
        status_counts.get(
            receipt[
                "status"
            ],
            0,
        )
        +
        1
    )


print(
    "Final condition statuses:"
)


for status, count in sorted(
    status_counts.items()
):

    print(
        f"  {status:30s}: {count}"
    )


print(
    "\n[PASS] 80 / 80 frozen conditions have durable final receipts."
)


# =============================================================================
# 11. BATCH-1 PREDICTION-INTEGRITY GATE
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-C :: BATCH-1 PREDICTION INTEGRITY"
)


preflight = json.loads(
    PREFLIGHT_RECEIPT.read_text(
        encoding="utf-8"
    )
)


expected_b1 = {
    (
        row[
            "target_id"
        ],
        row[
            "hardware_mode"
        ],
    ):
        row[
            "output_sha256"
        ]
    for row in preflight[
        "results"
    ]
}


b1_checked = 0


for receipt in all_receipts:

    if (
        receipt[
            "status"
        ] == "PASS"
        and
        int(
            receipt[
                "batch_size"
            ]
        ) == 1
    ):

        key = (
            receipt[
                "target_id"
            ],
            receipt[
                "hardware_mode"
            ],
        )

        expected_sha = expected_b1[
            key
        ]

        actual_sha = receipt[
            "timed_output_sha256"
        ]


        ok = (
            actual_sha
            ==
            expected_sha
        )


        print(
            f"{receipt['target_id']:43s} "
            f"{receipt['hardware_mode']:21s} "
            f"{'PASS' if ok else 'FAIL'}"
        )


        if not ok:

            raise RuntimeError(
                "Batch-1 prediction fingerprint changed."
            )


        b1_checked += 1


if b1_checked != 16:

    raise RuntimeError(
        f"Expected 16 batch-1 integrity checks; observed {b1_checked}."
    )


# =============================================================================
# 12. CREATE RESUME / INTERRUPTION PROVENANCE RECORD
# =============================================================================

resume_record = {
    "schema":
        "stage26_2_resume_execution_record_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-2",

    "recorded_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_HEAD,

    "original_interruption_after_durable_order":
        20,

    "isolated_recovery_condition":
        21,

    "condition_021_status":
        json.loads(
            (
                CONDITION_DIR
                / "condition_021.json"
            ).read_text(
                encoding="utf-8"
            )
        )[
            "status"
        ],

    "condition_021_receipt_sha256":
        EXPECTED_CONDITION_021_SHA256,

    "continuation_orders":
        [
            22,
            80,
        ],

    "parent_side_heartbeat_seconds":
        HEARTBEAT_SECONDS,

    "heartbeat_inside_measured_region":
        False,

    "frozen_worker_sha256":
        EXPECTED_WORKER_SHA256,

    "frozen_implementation_sha256":
        EXPECTED_IMPLEMENTATION_SHA256,

    "frozen_execution_plan_sha256":
        EXPECTED_PLAN_SHA256,

    "batch_policy_changed":
        False,

    "warmup_policy_changed":
        False,

    "timed_iteration_policy_changed":
        False,

    "thread_policy_changed":
        False,

    "affinity_policy_changed":
        False,

    "timeout_policy_changed":
        False,

    "model_changed":
        False,

    "input_generation_changed":
        False,

    "timing_boundary_changed":
        False,

    "completed_condition_remeasured":
        False,

    "memory_profiled":
        False,

    "gpu_used":
        False,

    "holdout_reopened":
        False,

    "final_condition_status_counts":
        status_counts,
}


atomic_json(
    RESUME_RECORD,
    resume_record,
)


resume_record_sha = sha256_file(
    RESUME_RECORD
)


print(
    "\nResume provenance SHA256:"
)

print(
    " ",
    resume_record_sha,
)


# =============================================================================
# 13. AGGREGATE RAW ITERATION TIMINGS
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-C :: RAW TIMING AGGREGATION"
)


raw_rows = []


for receipt in all_receipts:

    if receipt[
        "status"
    ] != "PASS":

        continue


    batch_size = int(
        receipt[
            "batch_size"
        ]
    )


    for iteration_index, elapsed_ns in enumerate(
        receipt[
            "elapsed_ns"
        ],
        start=1,
    ):

        elapsed_ns = int(
            elapsed_ns
        )

        seconds = (
            elapsed_ns
            /
            1e9
        )

        raw_rows.append(
            {
                "execution_order":
                    int(
                        receipt[
                            "execution_order"
                        ]
                    ),

                "condition_id":
                    receipt[
                        "condition_id"
                    ],

                "target_id":
                    receipt[
                        "target_id"
                    ],

                "target_role":
                    receipt[
                        "target_role"
                    ],

                "comparison_group":
                    receipt[
                        "comparison_group"
                    ],

                "hardware_mode":
                    receipt[
                        "hardware_mode"
                    ],

                "thread_count":
                    int(
                        receipt[
                            "thread_count"
                        ]
                    ),

                "batch_size":
                    batch_size,

                "iteration_index":
                    iteration_index,

                "elapsed_ns":
                    elapsed_ns,

                "amortized_ns_per_flow":
                    float(
                        elapsed_ns
                    )
                    /
                    float(
                        batch_size
                    ),

                "flows_per_second":
                    float(
                        batch_size
                    )
                    /
                    seconds,

                "status":
                    "PASS",
            }
        )


FULL_SUCCESS_DESIGN_COUNT = 8320


print(
    "Observed raw timings:",
    len(
        raw_rows
    ),
)

print(
    "Full-success design count:",
    FULL_SUCCESS_DESIGN_COUNT,
)


# =============================================================================
# 14. WRITE RAW JSONL + CSV
# =============================================================================

with RAW_JSONL.open(
    "w",
    encoding="utf-8",
) as f:

    for row in raw_rows:

        f.write(
            json.dumps(
                row,
                sort_keys=True,
            )
            +
            "\n"
        )


raw_columns = [
    "execution_order",
    "condition_id",
    "target_id",
    "target_role",
    "comparison_group",
    "hardware_mode",
    "thread_count",
    "batch_size",
    "iteration_index",
    "elapsed_ns",
    "amortized_ns_per_flow",
    "flows_per_second",
    "status",
]


with RAW_CSV.open(
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=raw_columns,
    )

    writer.writeheader()

    writer.writerows(
        raw_rows
    )


# =============================================================================
# 15. CONDITION STATUS TABLE
# =============================================================================

condition_status_rows = []


for receipt in all_receipts:

    elapsed_ns = np.asarray(
        receipt.get(
            "elapsed_ns",
            [],
        ),
        dtype=np.float64,
    )

    batch_size = int(
        receipt[
            "batch_size"
        ]
    )


    if (
        receipt[
            "status"
        ] == "PASS"
        and
        elapsed_ns.size > 0
    ):

        elapsed_ms = (
            elapsed_ns
            /
            1e6
        )

        throughput = (
            float(
                batch_size
            )
            /
            (
                elapsed_ns
                /
                1e9
            )
        )


        p50 = float(
            np.percentile(
                elapsed_ms,
                50,
            )
        )

        p95 = float(
            np.percentile(
                elapsed_ms,
                95,
            )
        )

        p99 = (
            float(
                np.percentile(
                    elapsed_ms,
                    99,
                )
            )
            if elapsed_ns.size >= 100
            else None
        )

        median_throughput = float(
            np.percentile(
                throughput,
                50,
            )
        )


    else:

        p50 = None
        p95 = None
        p99 = None
        median_throughput = None


    condition_status_rows.append(
        {
            "execution_order":
                int(
                    receipt[
                        "execution_order"
                    ]
                ),

            "condition_id":
                receipt[
                    "condition_id"
                ],

            "target_id":
                receipt[
                    "target_id"
                ],

            "target_role":
                receipt[
                    "target_role"
                ],

            "comparison_group":
                receipt[
                    "comparison_group"
                ],

            "hardware_mode":
                receipt[
                    "hardware_mode"
                ],

            "thread_count":
                int(
                    receipt[
                        "thread_count"
                    ]
                ),

            "batch_size":
                batch_size,

            "warmup_runs":
                int(
                    receipt[
                        "warmup_runs"
                    ]
                ),

            "timed_runs_planned":
                int(
                    receipt[
                        "timed_runs"
                    ]
                ),

            "timed_runs_observed":
                int(
                    elapsed_ns.size
                ),

            "status":
                receipt[
                    "status"
                ],

            "p50_batch_latency_ms":
                p50,

            "p95_batch_latency_ms":
                p95,

            "p99_batch_latency_ms":
                p99,

            "median_throughput_flows_per_second":
                median_throughput,
        }
    )


condition_status_rows = sorted(
    condition_status_rows,
    key=lambda row: int(
        row[
            "execution_order"
        ]
    ),
)


condition_status_columns = list(
    condition_status_rows[
        0
    ].keys()
)


with CONDITION_STATUS_CSV.open(
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=condition_status_columns,
    )

    writer.writeheader()

    writer.writerows(
        condition_status_rows
    )


# =============================================================================
# 16. STATISTICAL SUMMARY
# =============================================================================

banner(
    "STAGE26-2-RECOVERY-C :: STATISTICAL SUMMARY"
)


summary_rows = []


for receipt in all_receipts:

    if receipt[
        "status"
    ] != "PASS":

        continue


    elapsed_ns = np.asarray(
        receipt[
            "elapsed_ns"
        ],
        dtype=np.float64,
    )

    elapsed_ms = (
        elapsed_ns
        /
        1e6
    )

    batch_size = int(
        receipt[
            "batch_size"
        ]
    )

    throughput = (
        float(
            batch_size
        )
        /
        (
            elapsed_ns
            /
            1e9
        )
    )

    amortized_ms = (
        elapsed_ms
        /
        float(
            batch_size
        )
    )


    mean_ms = float(
        np.mean(
            elapsed_ms
        )
    )

    std_ms = float(
        np.std(
            elapsed_ms,
            ddof=1,
        )
    )

    p50_ms = float(
        np.percentile(
            elapsed_ms,
            50,
        )
    )

    p95_ms = float(
        np.percentile(
            elapsed_ms,
            95,
        )
    )

    p99_ms = (
        float(
            np.percentile(
                elapsed_ms,
                99,
            )
        )
        if elapsed_ms.size >= 100
        else None
    )


    seed_base = (
        receipt[
            "condition_id"
        ]
    )


    p50_ci = bootstrap_ci(
        elapsed_ms,
        lambda x: float(
            np.percentile(
                x,
                50,
            )
        ),
        seed=stable_seed(
            seed_base
            + "|p50"
        ),
    )


    p95_ci = bootstrap_ci(
        elapsed_ms,
        lambda x: float(
            np.percentile(
                x,
                95,
            )
        ),
        seed=stable_seed(
            seed_base
            + "|p95"
        ),
    )


    if elapsed_ms.size >= 100:

        p99_ci = bootstrap_ci(
            elapsed_ms,
            lambda x: float(
                np.percentile(
                    x,
                    99,
                )
            ),
            seed=stable_seed(
                seed_base
                + "|p99"
            ),
        )

    else:

        p99_ci = (
            None,
            None,
        )


    throughput_ci = bootstrap_ci(
        throughput,
        lambda x: float(
            np.percentile(
                x,
                50,
            )
        ),
        seed=stable_seed(
            seed_base
            + "|throughput"
        ),
    )


    summary_rows.append(
        {
            "execution_order":
                int(
                    receipt[
                        "execution_order"
                    ]
                ),

            "condition_id":
                receipt[
                    "condition_id"
                ],

            "target_id":
                receipt[
                    "target_id"
                ],

            "target_role":
                receipt[
                    "target_role"
                ],

            "comparison_group":
                receipt[
                    "comparison_group"
                ],

            "hardware_mode":
                receipt[
                    "hardware_mode"
                ],

            "thread_count":
                int(
                    receipt[
                        "thread_count"
                    ]
                ),

            "batch_size":
                batch_size,

            "n":
                int(
                    elapsed_ms.size
                ),

            "mean_batch_latency_ms":
                mean_ms,

            "std_batch_latency_ms":
                std_ms,

            "coefficient_of_variation":
                (
                    float(
                        std_ms
                        /
                        mean_ms
                    )
                    if mean_ms != 0
                    else None
                ),

            "p50_batch_latency_ms":
                p50_ms,

            "p95_batch_latency_ms":
                p95_ms,

            "p99_batch_latency_ms_if_n_gte_100":
                p99_ms,

            "maximum_batch_latency_ms_descriptive_only":
                float(
                    np.max(
                        elapsed_ms
                    )
                ),

            "p50_ci95_low_ms":
                p50_ci[
                    0
                ],

            "p50_ci95_high_ms":
                p50_ci[
                    1
                ],

            "p95_ci95_low_ms":
                p95_ci[
                    0
                ],

            "p95_ci95_high_ms":
                p95_ci[
                    1
                ],

            "p99_ci95_low_ms_if_n_gte_100":
                p99_ci[
                    0
                ],

            "p99_ci95_high_ms_if_n_gte_100":
                p99_ci[
                    1
                ],

            "median_amortized_latency_ms_per_flow":
                float(
                    np.percentile(
                        amortized_ms,
                        50,
                    )
                ),

            "median_throughput_flows_per_second":
                float(
                    np.percentile(
                        throughput,
                        50,
                    )
                ),

            "median_throughput_ci95_low":
                throughput_ci[
                    0
                ],

            "median_throughput_ci95_high":
                throughput_ci[
                    1
                ],
        }
    )


summary_rows = sorted(
    summary_rows,
    key=lambda row: int(
        row[
            "execution_order"
        ]
    ),
)


with SUMMARY_CSV.open(
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=list(
            summary_rows[
                0
            ].keys()
        ),
    )

    writer.writeheader()

    writer.writerows(
        summary_rows
    )


# =============================================================================
# 17. AGGREGATED CONDITION RECEIPTS
# =============================================================================

atomic_json(
    CONDITION_RECEIPTS_JSON,
    {
        "schema":
            "stage26_2_condition_receipts_v1",

        "stage":
            26,

        "checkpoint":
            "STAGE26-2",

        "parent_commit":
            EXPECTED_HEAD,

        "condition_count":
            80,

        "condition_receipts":
            all_receipts,
    },
)


# =============================================================================
# 18. SUMMARY JSON
# =============================================================================

atomic_json(
    SUMMARY_JSON,
    {
        "schema":
            "stage26_2_warm_cpu_summary_v1",

        "stage":
            26,

        "checkpoint":
            "STAGE26-2",

        "parent_commit":
            EXPECTED_HEAD,

        "measurement_protocol_sha256":
            EXPECTED_PROTOCOL_SHA256,

        "cpu_execution_plan_sha256":
            EXPECTED_PLAN_SHA256,

        "implementation_sha256":
            EXPECTED_IMPLEMENTATION_SHA256,

        "worker_sha256":
            EXPECTED_WORKER_SHA256,

        "resume_provenance_sha256":
            resume_record_sha,

        "condition_count":
            80,

        "condition_status_counts":
            status_counts,

        "raw_timing_observation_count":
            len(
                raw_rows
            ),

        "full_success_raw_timing_design_count":
            FULL_SUCCESS_DESIGN_COUNT,

        "bootstrap_replicates":
            BOOTSTRAP_REPLICATES,

        "summary_rows":
            summary_rows,

        "claim_boundary":
            (
                "Warm CPU model-inference latency and batch throughput only. "
                "Memory, feature extraction, end-to-end pipeline, GPU, "
                "Pareto, capacity and bottleneck conclusions remain pending."
            ),
    },
)


# =============================================================================
# 19. PRIMARY B=1 LATENCY SUMMARY
# =============================================================================

banner(
    "STAGE26-2 :: PRIMARY BATCH-1 CPU LATENCY SUMMARY"
)


batch1_rows = [
    row
    for row in summary_rows
    if int(
        row[
            "batch_size"
        ]
    ) == 1
]


for row in batch1_rows:

    p99 = row[
        "p99_batch_latency_ms_if_n_gte_100"
    ]

    p99_text = (
        f"{p99:.4f}"
        if p99 is not None
        else "NA"
    )


    print(
        f"{row['target_id']:43s} "
        f"{row['hardware_mode']:21s} "
        f"p50={row['p50_batch_latency_ms']:9.4f} ms "
        f"p95={row['p95_batch_latency_ms']:9.4f} ms "
        f"p99={p99_text:>9s} ms "
        f"median={row['median_throughput_flows_per_second']:11.1f} flow/s"
    )


# =============================================================================
# 20. CONDITION MATRIX
# =============================================================================

banner(
    "STAGE26-2 :: COMPLETE BATCH-SCALING MATRIX"
)


targets = sorted(
    {
        row[
            "target_id"
        ]
        for row in condition_status_rows
    }
)


for target in targets:

    print(
        "\n"
        + target
    )


    rows = [
        row
        for row in condition_status_rows
        if row[
            "target_id"
        ] == target
    ]


    rows = sorted(
        rows,
        key=lambda row: (
            row[
                "hardware_mode"
            ],
            int(
                row[
                    "batch_size"
                ]
            ),
        ),
    )


    for row in rows:

        if row[
            "status"
        ] == "PASS":

            print(
                f"  {row['hardware_mode']:21s} "
                f"B={int(row['batch_size']):5d} "
                f"PASS "
                f"p50={row['p50_batch_latency_ms']:11.3f} ms "
                f"median={row['median_throughput_flows_per_second']:12.1f} flow/s"
            )


        else:

            print(
                f"  {row['hardware_mode']:21s} "
                f"B={int(row['batch_size']):5d} "
                f"{row['status']}"
            )


# =============================================================================
# 21. FINAL STAGE26-2 RECEIPT
# =============================================================================

pass_count = int(
    status_counts.get(
        "PASS",
        0,
    )
)

oom_count = int(
    status_counts.get(
        "RESOURCE_LIMIT_OOM",
        0,
    )
)

timeout_count = int(
    status_counts.get(
        "TIMEOUT_RESOURCE_LIMIT",
        0,
    )
)


overall_status = (
    "PASS"
    if pass_count == 80
    else
    "COMPLETE_WITH_FROZEN_RESOURCE_LIMITS"
)


final_receipt = {
    "schema":
        "stage26_2_warm_cpu_receipt_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-2",

    "status":
        overall_status,

    "completed_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "cpu_execution_plan_sha256":
        EXPECTED_PLAN_SHA256,

    "cpu_preflight_receipt_sha256":
        EXPECTED_PREFLIGHT_SHA256,

    "cold_start_receipt_sha256":
        EXPECTED_COLD_SHA256,

    "implementation_sha256":
        EXPECTED_IMPLEMENTATION_SHA256,

    "worker_sha256":
        EXPECTED_WORKER_SHA256,

    "interruption_recovery_record_sha256":
        EXPECTED_RECOVERY_A_SHA256,

    "resume_execution_record_sha256":
        resume_record_sha,

    "condition_count":
        80,

    "pass_condition_count":
        pass_count,

    "resource_limit_oom_condition_count":
        oom_count,

    "timeout_resource_limit_condition_count":
        timeout_count,

    "raw_timing_observation_count":
        len(
            raw_rows
        ),

    "full_success_raw_timing_design_count":
        FULL_SUCCESS_DESIGN_COUNT,

    "raw_jsonl_sha256":
        sha256_file(
            RAW_JSONL
        ),

    "raw_csv_sha256":
        sha256_file(
            RAW_CSV
        ),

    "condition_receipts_sha256":
        sha256_file(
            CONDITION_RECEIPTS_JSON
        ),

    "condition_status_csv_sha256":
        sha256_file(
            CONDITION_STATUS_CSV
        ),

    "summary_csv_sha256":
        sha256_file(
            SUMMARY_CSV
        ),

    "summary_json_sha256":
        sha256_file(
            SUMMARY_JSON
        ),

    "warm_cpu_inference_measured":
        True,

    "batch_throughput_measured":
        True,

    "memory_profiled":
        False,

    "feature_extraction_measured":
        False,

    "end_to_end_pipeline_measured":
        False,

    "gpu_used":
        False,

    "holdout_reopened":
        False,

    "pareto_analysis_performed":
        False,

    "capacity_analysis_performed":
        False,

    "operator_interruption_occurred":
        True,

    "completed_conditions_remeasured_after_interruption":
        False,

    "scientific_measurement_policy_changed_after_interruption":
        False,

    "claim_boundary":
        (
            "Warm CPU inference and batch throughput only. "
            "No memory, extraction, end-to-end, GPU, bottleneck, "
            "capacity, or Pareto conclusion is permitted yet."
        ),

    "next_action":
        "GIT_ANCHOR_STAGE26_2_BEFORE_MEMORY_PROFILING",
}


atomic_json(
    RECEIPT_PATH,
    final_receipt,
)


receipt_sha = sha256_file(
    RECEIPT_PATH
)


print(
    "\nFinal Stage26-2 status:",
    overall_status,
)

print(
    "Receipt SHA256:"
)

print(
    " ",
    receipt_sha,
)


# =============================================================================
# 22. BUILD DURABLE REPOSITORY PACKAGE
# =============================================================================

banner(
    "STAGE26-2 :: BUILD DURABLE REPOSITORY PACKAGE"
)


if REPO_STAGE26_2.exists():

    raise RuntimeError(
        "Stage26-2 repository package unexpectedly already exists."
    )


REPO_STAGE26_2.mkdir(
    parents=True,
    exist_ok=False,
)


# Core aggregate artifacts.
files_to_copy = [
    WORKER,
    IMPLEMENTATION,
    RECOVERY_A_RECORD,
    RESUME_RECORD,
    CONDITION_RECEIPTS_JSON,
    CONDITION_STATUS_CSV,
    RAW_JSONL,
    RAW_CSV,
    SUMMARY_CSV,
    SUMMARY_JSON,
    RECEIPT_PATH,
]


for source in files_to_copy:

    shutil.copy2(
        source,
        REPO_STAGE26_2
        / source.name,
    )

    print(
        "[COPIED]",
        source.name,
    )


# Preserve all 80 original condition receipts as provenance.
repo_condition_dir = (
    REPO_STAGE26_2
    / "condition_receipts"
)


repo_condition_dir.mkdir(
    parents=True,
    exist_ok=False,
)


for order in range(
    1,
    81,
):

    source = (
        CONDITION_DIR
        / f"condition_{order:03d}.json"
    )

    shutil.copy2(
        source,
        repo_condition_dir
        / source.name,
    )


print(
    "[COPIED] 80 individual condition receipts"
)


# =============================================================================
# 23. PACKAGE MANIFEST
# =============================================================================

PACKAGE_MANIFEST = (
    REPO_STAGE26_2
    / "stage26_2_warm_cpu_package_manifest.json"
)


package_files = []


for path in sorted(
    REPO_STAGE26_2.rglob(
        "*"
    )
):

    if (
        path.is_file()
        and
        path != PACKAGE_MANIFEST
    ):

        package_files.append(
            {
                "path":
                    str(
                        path.relative_to(
                            REPO
                        )
                    ),

                "size_bytes":
                    int(
                        path.stat().st_size
                    ),

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )


package_manifest = {
    "schema":
        "stage26_2_warm_cpu_package_manifest_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-2",

    "status":
        "READY_FOR_GIT_ANCHOR",

    "parent_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "receipt_sha256":
        receipt_sha,

    "interruption_recovery_record_sha256":
        EXPECTED_RECOVERY_A_SHA256,

    "resume_execution_record_sha256":
        resume_record_sha,

    "individual_condition_receipt_count":
        80,

    "file_count_excluding_manifest":
        len(
            package_files
        ),

    "files":
        package_files,

    "scientific_boundary":
        {
            "warm_cpu_inference_measured":
                True,

            "batch_throughput_measured":
                True,

            "memory_profiled":
                False,

            "feature_extraction_measured":
                False,

            "end_to_end_pipeline_measured":
                False,

            "gpu_used":
                False,

            "holdout_reopened":
                False,

            "pareto_analysis_performed":
                False,

            "operator_interruption_occurred":
                True,

            "completed_conditions_remeasured":
                False,
        },
}


atomic_json(
    PACKAGE_MANIFEST,
    package_manifest,
)


package_sha = sha256_file(
    PACKAGE_MANIFEST
)


# =============================================================================
# 24. FINAL REPOSITORY AUDIT
# =============================================================================

banner(
    "STAGE26-2 FINAL AUDIT"
)


repo_status_after = git(
    "status",
    "--porcelain",
)


print(
    "Repository status:"
)

print(
    repo_status_after
    if repo_status_after
    else "<unexpected clean>"
)


if not repo_status_after:

    raise RuntimeError(
        "Expected new Stage26-2 repository package."
    )


unexpected = []


for line in repo_status_after.splitlines():

    relpath = line[
        3:
    ]


    if not relpath.startswith(
        "results/stage26_deployment_profiling/"
        "stage26_2_cpu_warm_inference/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected changes outside Stage26-2:\n"
        +
        "\n".join(
            unexpected
        )
    )


# Upstream identities remain immutable.
for name, path, expected_sha in immutable_checks[:6]:

    if sha256_file(
        path
    ) != expected_sha:

        raise RuntimeError(
            f"Upstream frozen artifact changed during continuation: {name}"
        )


# =============================================================================
# 25. CLOSURE
# =============================================================================

banner(
    "STAGE26-2 WARM CPU PROFILING COMPLETE"
)


print(
    "PARENT COMMIT:"
)

print(
    " ",
    EXPECTED_HEAD,
)


print(
    "\nFROZEN CONDITIONS:"
)

print(
    "  80 / 80 FINAL RECEIPTS"
)


print(
    "\nSTATUS COUNTS:"
)

print(
    "  PASS                   :",
    pass_count,
)

print(
    "  RESOURCE_LIMIT_OOM     :",
    oom_count,
)

print(
    "  TIMEOUT_RESOURCE_LIMIT :",
    timeout_count,
)


print(
    "\nRAW TIMED OBSERVATIONS:"
)

print(
    " ",
    len(
        raw_rows
    ),
)

print(
    "  full-success design =",
    FULL_SUCCESS_DESIGN_COUNT,
)


print(
    "\nRAW JSONL SHA256:"
)

print(
    " ",
    sha256_file(
        RAW_JSONL
    ),
)


print(
    "\nRAW CSV SHA256:"
)

print(
    " ",
    sha256_file(
        RAW_CSV
    ),
)


print(
    "\nSUMMARY CSV SHA256:"
)

print(
    " ",
    sha256_file(
        SUMMARY_CSV
    ),
)


print(
    "\nINTERRUPTION RECORD SHA256:"
)

print(
    " ",
    EXPECTED_RECOVERY_A_SHA256,
)


print(
    "\nRESUME RECORD SHA256:"
)

print(
    " ",
    resume_record_sha,
)


print(
    "\nRECEIPT SHA256:"
)

print(
    " ",
    receipt_sha,
)


print(
    "\nPACKAGE MANIFEST SHA256:"
)

print(
    " ",
    package_sha,
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  WARM CPU INFERENCE LATENCY COMPLETE"
)

print(
    "  CPU BATCH THROUGHPUT COMPLETE"
)

print(
    "  RAW PER-ITERATION NANOSECONDS RETAINED"
)

print(
    "  ALL 80 CONDITION RECEIPTS PRESERVED"
)

print(
    "  OPERATOR INTERRUPTION TRANSPARENTLY RECORDED"
)

print(
    "  CONDITIONS 001..020 WERE NEVER REMEASURED"
)

print(
    "  CONDITION 021 EXECUTED EXACTLY ONCE AFTER INTERRUPTION"
)

print(
    "  CONDITIONS 022..080 EXECUTED/RESUMED IN FROZEN ORDER"
)

print(
    "  NO BATCH SUBSTITUTION"
)

print(
    "  NO ITERATION SUBSTITUTION"
)

print(
    "  NO TIMEOUT CHANGE"
)

print(
    "  MEMORY NOT YET PROFILED"
)

print(
    "  EXTRACTION NOT YET PROFILED"
)

print(
    "  END-TO-END PIPELINE NOT YET PROFILED"
)

print(
    "  PARETO NOT YET PERFORMED"
)

print(
    "  GPU NOT USED"
)

print(
    "  HOLDOUT NOT REOPENED"
)


print(
    "\nCRITICAL NEXT ACTION:"
)

print(
    "  STAGE26-2-GIT — commit, push, and remotely verify the complete"
)

print(
    "  warm CPU package BEFORE Stage26-3 memory/package profiling."
)


STAGE26-2-RECOVERY-C :: SCIENTIFIC / IMMUTABILITY GATE
Expected HEAD : 46379b6d036008db4d60b056a66f4c01383e3298
Local HEAD    : 46379b6d036008db4d60b056a66f4c01383e3298
origin/main   : 46379b6d036008db4d60b056a66f4c01383e3298
Repository clean: True
measurement protocol               PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
CPU execution plan                 PASS b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363
CPU preflight receipt              PASS 4c7fc55a0e44e53587d385989dfec4f3f57f20768c7dd4a2d08e2936d2ba21e5
cold-start receipt                 PASS 35d988fbdcf7d84d54e593cd37f1204a005c2e854ae7128cc559629616d99c10
warm CPU worker                    PASS 2248f9c61b896bccf306167524b8be978993f49fc198327a6e280ee4553709b4
warm implementation                PASS 678c53993799f66e1528808f46c9c27c64b8821dcb8de2ce83343f126f848d43
interruption recovery record       PASS 55e95ccc3426f52379931ff1126e4a6d8dfbf06ca75cd1c90dda18c078c6dcf8
condition 021 r

In [20]:
# =============================================================================
# STAGE26-2-GIT — ANCHOR COMPLETE WARM CPU PROFILING RESULTS
#
# PARENT:
#   46379b6d036008db4d60b056a66f4c01383e3298
#
# EXPECTED SCIENTIFIC RESULT:
#   80 / 80 frozen conditions final
#   73 PASS
#   2 RESOURCE_LIMIT_OOM
#   5 TIMEOUT_RESOURCE_LIMIT
#   8120 raw timed observations
#
# THIS CELL ONLY:
#   - verifies Stage26-2 local package
#   - verifies package manifest and every manifest-listed file
#   - verifies recovery provenance
#   - commits results
#   - pushes main
#   - independently verifies remote ref
#   - verifies every Stage26-2 file byte-for-byte from origin/main
#   - re-verifies all upstream Stage26 locks
#
# THIS CELL DOES NOT:
#   - load a model
#   - run inference
#   - collect timing
#   - profile memory
#   - access holdout
#   - use GPU
#
# IDEMPOTENT:
#   Accidental second execution verifies the existing Stage26-2 commit.
# =============================================================================

from __future__ import annotations

import os
import json
import stat
import hashlib
import tempfile
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient


# =============================================================================
# 0. FROZEN IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "46379b6d036008db4d60b056a66f4c01383e3298"
)

COMMIT_MESSAGE = (
    "stage26: anchor warm CPU deployment profiling"
)


STAGE26_2_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_2_cpu_warm_inference"
)

MANIFEST_REL = (
    STAGE26_2_REL
    + "/stage26_2_warm_cpu_package_manifest.json"
)

RECEIPT_REL = (
    STAGE26_2_REL
    + "/stage26_2_warm_cpu_receipt.json"
)

RAW_JSONL_REL = (
    STAGE26_2_REL
    + "/stage26_2_warm_raw.jsonl"
)

RAW_CSV_REL = (
    STAGE26_2_REL
    + "/stage26_2_warm_raw.csv"
)

SUMMARY_CSV_REL = (
    STAGE26_2_REL
    + "/stage26_2_warm_summary.csv"
)

INTERRUPTION_REL = (
    STAGE26_2_REL
    + "/stage26_2_interruption_recovery_record.json"
)

RESUME_REL = (
    STAGE26_2_REL
    + "/stage26_2_resume_execution_record.json"
)


EXPECTED_MANIFEST_SHA256 = (
    "7f4d01bb3fe685dca528a4a1c788bc5819b42a4e3080098fc3e923b2fb0e0f19"
)

EXPECTED_RECEIPT_SHA256 = (
    "b4b2623eabde7dd6b9acc250357a9f1ad61fa342c1dd496dcb634d4bfaca4d15"
)

EXPECTED_RAW_JSONL_SHA256 = (
    "64e7961dc8a37c00ec08a34f77a6d50a8a5ec352ff1b75683360abc1b7147438"
)

EXPECTED_RAW_CSV_SHA256 = (
    "78c58289ccfc4598966d6516201028f43c4b1d16d8bd0268a0cf58129d4179fa"
)

EXPECTED_SUMMARY_CSV_SHA256 = (
    "75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232"
)

EXPECTED_INTERRUPTION_SHA256 = (
    "55e95ccc3426f52379931ff1126e4a6d8dfbf06ca75cd1c90dda18c078c6dcf8"
)

EXPECTED_RESUME_SHA256 = (
    "a28cac34a5dfd45db7a172ee30cbfe2efa91a8f75dfa127b46eb41f2dd42a2e3"
)


# -------------------------------------------------------------------------
# Upstream immutable Stage26 anchors
# -------------------------------------------------------------------------

PROTOCOL_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/"
    "measurement_protocol.json"
)

PLAN_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/"
    "cpu_execution_plan.json"
)

PREFLIGHT_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0d_cpu_preflight/"
    "stage26_0d_cpu_preflight_receipt.json"
)

COLD_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_1_cpu_cold_start/"
    "stage26_1_cold_start_receipt.json"
)


EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

EXPECTED_PLAN_SHA256 = (
    "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363"
)

EXPECTED_PREFLIGHT_SHA256 = (
    "4c7fc55a0e44e53587d385989dfec4f3f57f20768c7dd4a2d08e2936d2ba21e5"
)

EXPECTED_COLD_SHA256 = (
    "35d988fbdcf7d84d54e593cd37f1204a005c2e854ae7128cc559629616d99c10"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):
    print(
        "\n"
        + "=" * 112
    )
    print(text)
    print(
        "=" * 112
    )


def run(
    cmd,
    *,
    env=None,
    check=True,
):
    p = subprocess.run(
        cmd,
        cwd=REPO,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "$ "
            + " ".join(cmd)
            + "\n\n"
            + p.stdout
        )

    return p


def git(
    *args,
    env=None,
    check=True,
):
    return run(
        [
            "git",
            *args,
        ],
        env=env,
        check=check,
    ).stdout.strip()


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def sha256_bytes(data):
    return hashlib.sha256(
        data
    ).hexdigest()


def git_blob_bytes(
    ref,
    relpath,
):

    p = subprocess.run(
        [
            "git",
            "show",
            f"{ref}:{relpath}",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stderr.decode(
                "utf-8",
                errors="replace",
            )
        )

    return p.stdout


# =============================================================================
# 2. TOP-LEVEL PACKAGE HASH GATE
# =============================================================================

banner(
    "STAGE26-2-GIT :: LOCAL TOP-LEVEL HASH GATE"
)


top_checks = [
    (
        "package manifest",
        MANIFEST_REL,
        EXPECTED_MANIFEST_SHA256,
    ),
    (
        "warm CPU receipt",
        RECEIPT_REL,
        EXPECTED_RECEIPT_SHA256,
    ),
    (
        "raw JSONL",
        RAW_JSONL_REL,
        EXPECTED_RAW_JSONL_SHA256,
    ),
    (
        "raw CSV",
        RAW_CSV_REL,
        EXPECTED_RAW_CSV_SHA256,
    ),
    (
        "summary CSV",
        SUMMARY_CSV_REL,
        EXPECTED_SUMMARY_CSV_SHA256,
    ),
    (
        "interruption record",
        INTERRUPTION_REL,
        EXPECTED_INTERRUPTION_SHA256,
    ),
    (
        "resume record",
        RESUME_REL,
        EXPECTED_RESUME_SHA256,
    ),
]


for name, relpath, expected in top_checks:

    path = (
        REPO
        / relpath
    )

    if not path.exists():

        raise FileNotFoundError(
            path
        )


    actual = sha256_file(
        path
    )

    ok = (
        actual
        ==
        expected
    )


    print(
        f"{name:24s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual}"
    )


    if not ok:

        raise RuntimeError(
            f"Stage26-2 local SHA mismatch: {name}"
        )


# =============================================================================
# 3. MANIFEST-COMPLETE LOCAL VERIFICATION
# =============================================================================

banner(
    "STAGE26-2-GIT :: MANIFEST-COMPLETE LOCAL GATE"
)


manifest_path = (
    REPO
    / MANIFEST_REL
)


manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)


if manifest.get(
    "status"
) != "READY_FOR_GIT_ANCHOR":

    raise RuntimeError(
        "Stage26-2 package is not READY_FOR_GIT_ANCHOR."
    )


if manifest.get(
    "parent_commit"
) != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-2 manifest parent commit mismatch."
    )


if int(
    manifest.get(
        "individual_condition_receipt_count",
        -1,
    )
) != 80:

    raise RuntimeError(
        "Stage26-2 manifest does not preserve exactly 80 condition receipts."
    )


manifest_files = manifest[
    "files"
]


print(
    "Manifest-listed files:",
    len(
        manifest_files
    ),
)


for record in manifest_files:

    relpath = record[
        "path"
    ]

    path = (
        REPO
        / relpath
    )

    expected_sha = record[
        "sha256"
    ]

    expected_size = int(
        record[
            "size_bytes"
        ]
    )


    if not path.exists():

        raise FileNotFoundError(
            path
        )


    actual_sha = sha256_file(
        path
    )

    actual_size = int(
        path.stat().st_size
    )


    sha_ok = (
        actual_sha
        ==
        expected_sha
    )

    size_ok = (
        actual_size
        ==
        expected_size
    )


    print(
        f"{relpath:105s} "
        f"sha={'PASS' if sha_ok else 'FAIL'} "
        f"size={'PASS' if size_ok else 'FAIL'}"
    )


    if not sha_ok or not size_ok:

        raise RuntimeError(
            f"Manifest verification failed: {relpath}"
        )


# =============================================================================
# 4. SCIENTIFIC RECEIPT GATE
# =============================================================================

banner(
    "STAGE26-2-GIT :: SCIENTIFIC RECEIPT GATE"
)


receipt = json.loads(
    (
        REPO
        / RECEIPT_REL
    ).read_text(
        encoding="utf-8"
    )
)


expected_receipt_fields = {
    "status":
        "COMPLETE_WITH_FROZEN_RESOURCE_LIMITS",

    "condition_count":
        80,

    "pass_condition_count":
        73,

    "resource_limit_oom_condition_count":
        2,

    "timeout_resource_limit_condition_count":
        5,

    "raw_timing_observation_count":
        8120,

    "full_success_raw_timing_design_count":
        8320,

    "warm_cpu_inference_measured":
        True,

    "batch_throughput_measured":
        True,

    "memory_profiled":
        False,

    "feature_extraction_measured":
        False,

    "end_to_end_pipeline_measured":
        False,

    "gpu_used":
        False,

    "holdout_reopened":
        False,

    "pareto_analysis_performed":
        False,

    "capacity_analysis_performed":
        False,

    "operator_interruption_occurred":
        True,

    "completed_conditions_remeasured_after_interruption":
        False,

    "scientific_measurement_policy_changed_after_interruption":
        False,
}


for key, expected in expected_receipt_fields.items():

    actual = receipt.get(
        key
    )

    print(
        f"{key:52s}: {actual!r}"
    )


    if actual != expected:

        raise RuntimeError(
            f"Stage26-2 receipt mismatch: "
            f"{key}={actual!r}, expected {expected!r}"
        )


if receipt.get(
    "parent_commit"
) != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-2 receipt parent mismatch."
    )


print(
    "\n[PASS] Scientific boundary and resource-limit outcomes verified."
)


# =============================================================================
# 5. VERIFY ALL 80 INDIVIDUAL CONDITION RECEIPTS ARE PRESENT
# =============================================================================

banner(
    "STAGE26-2-GIT :: 80 INDIVIDUAL CONDITION RECEIPTS"
)


condition_dir = (
    REPO
    / STAGE26_2_REL
    / "condition_receipts"
)


condition_paths = sorted(
    condition_dir.glob(
        "condition_*.json"
    )
)


print(
    "Condition receipt count:",
    len(
        condition_paths
    ),
)


if len(
    condition_paths
) != 80:

    raise RuntimeError(
        "Expected exactly 80 committed-package condition receipts."
    )


orders = []


status_counts = {}


for path in condition_paths:

    row = json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


    order = int(
        row[
            "execution_order"
        ]
    )

    orders.append(
        order
    )


    status = row[
        "status"
    ]


    status_counts[
        status
    ] = (
        status_counts.get(
            status,
            0,
        )
        +
        1
    )


if sorted(
    orders
) != list(
    range(
        1,
        81,
    )
):

    raise RuntimeError(
        "Individual receipt execution orders are not exactly 1..80."
    )


print(
    "PASS                   :",
    status_counts.get(
        "PASS",
        0,
    )
)

print(
    "RESOURCE_LIMIT_OOM     :",
    status_counts.get(
        "RESOURCE_LIMIT_OOM",
        0,
    )
)

print(
    "TIMEOUT_RESOURCE_LIMIT :",
    status_counts.get(
        "TIMEOUT_RESOURCE_LIMIT",
        0,
    )
)


if status_counts != {
    "PASS":
        73,

    "RESOURCE_LIMIT_OOM":
        2,

    "TIMEOUT_RESOURCE_LIMIT":
        5,
}:

    raise RuntimeError(
        f"Unexpected individual receipt status counts: {status_counts}"
    )


# =============================================================================
# 6. CURRENT GIT STATE / IDEMPOTENT COMMIT LOGIC
# =============================================================================

banner(
    "STAGE26-2-GIT :: CURRENT GIT STATE"
)


HEAD_BEFORE = git(
    "rev-parse",
    "HEAD",
)

STATUS_BEFORE = git(
    "status",
    "--porcelain",
)


print(
    "Current HEAD:",
    HEAD_BEFORE,
)

print(
    "Repository clean:",
    STATUS_BEFORE == "",
)


if STATUS_BEFORE:

    print(
        "\nRepository status:"
    )

    print(
        STATUS_BEFORE
    )


already_committed = False


if HEAD_BEFORE == EXPECTED_PARENT:

    already_committed = False


else:

    parent = git(
        "rev-parse",
        "HEAD^",
        check=False,
    )

    subject = git(
        "log",
        "-1",
        "--pretty=%s",
    )


    if (
        parent
        ==
        EXPECTED_PARENT
        and
        subject
        ==
        COMMIT_MESSAGE
    ):

        already_committed = True

        print(
            "\n[INFO] Stage26-2 anchor commit already exists locally."
        )


    else:

        raise RuntimeError(
            "Current HEAD is neither the Stage26-1 parent nor "
            "the expected idempotent Stage26-2 commit."
        )


# =============================================================================
# 7. STAGE + COMMIT IF NEEDED
# =============================================================================

if not already_committed:

    banner(
        "STAGE26-2-GIT :: STAGE + COMMIT"
    )


    if not STATUS_BEFORE:

        raise RuntimeError(
            "Expected uncommitted Stage26-2 package, but worktree is clean."
        )


    unexpected = []


    for line in STATUS_BEFORE.splitlines():

        relpath = line[
            3:
        ]


        if not relpath.startswith(
            STAGE26_2_REL
            + "/"
        ):

            unexpected.append(
                line
            )


    if unexpected:

        raise RuntimeError(
            "Unexpected repository modifications outside Stage26-2:\n"
            +
            "\n".join(
                unexpected
            )
        )


    git(
        "add",
        "--",
        STAGE26_2_REL,
    )


    staged = git(
        "diff",
        "--cached",
        "--name-status",
    )


    print(
        staged
    )


    if not staged:

        raise RuntimeError(
            "Nothing staged for Stage26-2."
        )


    for line in staged.splitlines():

        relpath = line.split(
            "\t"
        )[
            -1
        ]


        if not relpath.startswith(
            STAGE26_2_REL
            + "/"
        ):

            raise RuntimeError(
                "Unexpected staged path:\n"
                + line
            )


    if not git(
        "config",
        "--get",
        "user.name",
        check=False,
    ):

        git(
            "config",
            "user.name",
            "themubasshir",
        )


    if not git(
        "config",
        "--get",
        "user.email",
        check=False,
    ):

        git(
            "config",
            "user.email",
            "themubasshir@users.noreply.github.com",
        )


    commit_output = git(
        "commit",
        "-m",
        COMMIT_MESSAGE,
    )


    print(
        "\n"
        + commit_output
    )


    STAGE26_2_HEAD = git(
        "rev-parse",
        "HEAD",
    )


    commit_parent = git(
        "rev-parse",
        "HEAD^",
    )


    if commit_parent != EXPECTED_PARENT:

        raise RuntimeError(
            "Stage26-2 commit parent mismatch."
        )


else:

    STAGE26_2_HEAD = HEAD_BEFORE


print(
    "\nStage26-2 commit:",
    STAGE26_2_HEAD,
)


# =============================================================================
# 8. COMMITTED-BLOB HASH VERIFICATION
# =============================================================================

banner(
    "STAGE26-2-GIT :: COMMITTED-BLOB HASH GATE"
)


manifest_blob_sha = sha256_bytes(
    git_blob_bytes(
        "HEAD",
        MANIFEST_REL,
    )
)


print(
    f"package manifest "
    f"{'PASS' if manifest_blob_sha == EXPECTED_MANIFEST_SHA256 else 'FAIL'} "
    f"{manifest_blob_sha}"
)


if manifest_blob_sha != EXPECTED_MANIFEST_SHA256:

    raise RuntimeError(
        "Committed Stage26-2 manifest mismatch."
    )


for record in manifest_files:

    relpath = record[
        "path"
    ]

    expected_sha = record[
        "sha256"
    ]


    committed_sha = sha256_bytes(
        git_blob_bytes(
            "HEAD",
            relpath,
        )
    )


    ok = (
        committed_sha
        ==
        expected_sha
    )


    print(
        f"{relpath:105s} "
        f"{'PASS' if ok else 'FAIL'}"
    )


    if not ok:

        raise RuntimeError(
            f"Committed blob mismatch: {relpath}"
        )


# =============================================================================
# 9. FETCH CURRENT REMOTE RELATIONSHIP
# =============================================================================

banner(
    "STAGE26-2-GIT :: REMOTE RELATIONSHIP"
)


git(
    "fetch",
    "origin",
    "main",
)


REMOTE_BEFORE = git(
    "rev-parse",
    "origin/main",
)


print(
    "Local Stage26-2 HEAD:",
    STAGE26_2_HEAD,
)

print(
    "origin/main currently:",
    REMOTE_BEFORE,
)


if REMOTE_BEFORE == STAGE26_2_HEAD:

    relationship = (
        "REMOTE_ALREADY_HAS_STAGE26_2"
    )


elif REMOTE_BEFORE == EXPECTED_PARENT:

    relationship = (
        "LOCAL_STAGE26_2_AHEAD_BY_ONE"
    )


else:

    relationship = (
        "UNEXPECTED_REMOTE_DIVERGENCE"
    )


print(
    "Relationship:",
    relationship,
)


if relationship == "UNEXPECTED_REMOTE_DIVERGENCE":

    raise RuntimeError(
        "origin/main diverged unexpectedly. Do NOT force push."
    )


# =============================================================================
# 10. PUSH IF NEEDED
# =============================================================================

if relationship == "LOCAL_STAGE26_2_AHEAD_BY_ONE":

    banner(
        "STAGE26-2-GIT :: PUSH"
    )


    secret_client = (
        UserSecretsClient()
    )


    aliases = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "gh_token",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
        "gh_pat",
    ]


    token = None

    token_label = None


    for label in aliases:

        try:

            value = (
                secret_client.get_secret(
                    label
                )
            )

        except Exception:

            value = None


        if value:

            token = (
                value.strip()
            )

            token_label = label

            break


    if not token:

        raise RuntimeError(
            "No usable GitHub credential found in Kaggle Secrets."
        )


    print(
        f"[FOUND] {token_label} "
        f"({len(token)} characters)"
    )

    print(
        "Token value will not be printed."
    )


    fd, askpass_name = tempfile.mkstemp(
        prefix="stage26_2_askpass_",
        suffix=".sh",
    )

    os.close(
        fd
    )


    askpass = Path(
        askpass_name
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' "x-access-token" ;;
  *Password*) printf '%s\\n' "$GITHUB_TOKEN" ;;
  *)          printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        stat.S_IRUSR
        |
        stat.S_IWUSR
        |
        stat.S_IXUSR
    )


    push_env = (
        os.environ.copy()
    )


    push_env[
        "GITHUB_TOKEN"
    ] = token

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"


    try:

        result = run(
            [
                "git",
                "push",
                "origin",
                "HEAD:main",
            ],
            env=push_env,
        )


        print(
            result.stdout
        )


    finally:

        try:
            askpass.unlink()

        except FileNotFoundError:
            pass


        token = None


# =============================================================================
# 11. FETCH BACK / REF VERIFICATION
# =============================================================================

banner(
    "STAGE26-2-GIT :: REMOTE COMMIT GATE"
)


git(
    "fetch",
    "origin",
    "main",
)


LOCAL_HEAD = git(
    "rev-parse",
    "HEAD",
)

REMOTE_HEAD = git(
    "rev-parse",
    "origin/main",
)


print(
    "Local HEAD :",
    LOCAL_HEAD,
)

print(
    "Remote HEAD:",
    REMOTE_HEAD,
)


if LOCAL_HEAD != STAGE26_2_HEAD:

    raise RuntimeError(
        "Local HEAD changed unexpectedly."
    )


if REMOTE_HEAD != STAGE26_2_HEAD:

    raise RuntimeError(
        "origin/main does not match Stage26-2 commit."
    )


ls_remote_output = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


ls_remote_sha = (
    ls_remote_output.split()[0]
    if ls_remote_output
    else None
)


print(
    "ls-remote :",
    ls_remote_sha,
)


if ls_remote_sha != STAGE26_2_HEAD:

    raise RuntimeError(
        "Independent remote ref verification failed."
    )


# =============================================================================
# 12. BYTE-FOR-BYTE REMOTE PACKAGE VERIFICATION
# =============================================================================

banner(
    "STAGE26-2-GIT :: REMOTE PACKAGE HASH GATE"
)


remote_manifest_sha = sha256_bytes(
    git_blob_bytes(
        "origin/main",
        MANIFEST_REL,
    )
)


manifest_ok = (
    remote_manifest_sha
    ==
    EXPECTED_MANIFEST_SHA256
)


print(
    f"package manifest "
    f"{'PASS' if manifest_ok else 'FAIL'} "
    f"{remote_manifest_sha}"
)


if not manifest_ok:

    raise RuntimeError(
        "Remote Stage26-2 manifest mismatch."
    )


remote_pass_count = 0


for index, record in enumerate(
    manifest_files,
    start=1,
):

    relpath = record[
        "path"
    ]

    expected_sha = record[
        "sha256"
    ]


    remote_sha = sha256_bytes(
        git_blob_bytes(
            "origin/main",
            relpath,
        )
    )


    ok = (
        remote_sha
        ==
        expected_sha
    )


    if not ok:

        raise RuntimeError(
            f"Remote Stage26-2 file mismatch: {relpath}"
        )


    remote_pass_count += 1


    print(
        f"[{index:03d}/{len(manifest_files):03d}] "
        f"PASS {relpath}"
    )


print(
    "\nRemote manifest-listed files verified:",
    remote_pass_count,
)


# =============================================================================
# 13. EXPLICIT PRIMARY RESULT REMOTE HASHES
# =============================================================================

banner(
    "STAGE26-2-GIT :: PRIMARY RESULT REMOTE HASHES"
)


primary_checks = [
    (
        "receipt",
        RECEIPT_REL,
        EXPECTED_RECEIPT_SHA256,
    ),
    (
        "raw JSONL",
        RAW_JSONL_REL,
        EXPECTED_RAW_JSONL_SHA256,
    ),
    (
        "raw CSV",
        RAW_CSV_REL,
        EXPECTED_RAW_CSV_SHA256,
    ),
    (
        "summary CSV",
        SUMMARY_CSV_REL,
        EXPECTED_SUMMARY_CSV_SHA256,
    ),
    (
        "interruption",
        INTERRUPTION_REL,
        EXPECTED_INTERRUPTION_SHA256,
    ),
    (
        "resume record",
        RESUME_REL,
        EXPECTED_RESUME_SHA256,
    ),
]


for name, relpath, expected in primary_checks:

    remote_sha = sha256_bytes(
        git_blob_bytes(
            "origin/main",
            relpath,
        )
    )


    ok = (
        remote_sha
        ==
        expected
    )


    print(
        f"{name:18s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{remote_sha}"
    )


    if not ok:

        raise RuntimeError(
            f"Remote primary result mismatch: {name}"
        )


# =============================================================================
# 14. UPSTREAM IMMUTABILITY GATE
# =============================================================================

banner(
    "STAGE26-2-GIT :: UPSTREAM IMMUTABILITY GATE"
)


upstream_checks = [
    (
        "measurement_protocol",
        PROTOCOL_REL,
        EXPECTED_PROTOCOL_SHA256,
    ),
    (
        "cpu_execution_plan",
        PLAN_REL,
        EXPECTED_PLAN_SHA256,
    ),
    (
        "cpu_preflight_receipt",
        PREFLIGHT_REL,
        EXPECTED_PREFLIGHT_SHA256,
    ),
    (
        "cold_start_receipt",
        COLD_REL,
        EXPECTED_COLD_SHA256,
    ),
]


for name, relpath, expected in upstream_checks:

    remote_sha = sha256_bytes(
        git_blob_bytes(
            "origin/main",
            relpath,
        )
    )


    ok = (
        remote_sha
        ==
        expected
    )


    print(
        f"{name:28s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{remote_sha}"
    )


    if not ok:

        raise RuntimeError(
            f"Upstream frozen Stage26 artifact changed: {name}"
        )


# =============================================================================
# 15. FINAL CLEANLINESS / COMMIT STRUCTURE
# =============================================================================

banner(
    "STAGE26-2-GIT :: FINAL AUDIT"
)


FINAL_STATUS = git(
    "status",
    "--porcelain",
)


print(
    "Repository clean:",
    FINAL_STATUS == "",
)


if FINAL_STATUS:

    print(
        FINAL_STATUS
    )

    raise RuntimeError(
        "Repository is dirty after Stage26-2 anchor."
    )


FINAL_PARENT = git(
    "rev-parse",
    "HEAD^",
)


if FINAL_PARENT != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-2 anchored commit parent mismatch."
    )


FINAL_SUBJECT = git(
    "log",
    "-1",
    "--pretty=%s",
)


if FINAL_SUBJECT != COMMIT_MESSAGE:

    raise RuntimeError(
        "Stage26-2 commit message mismatch."
    )


# =============================================================================
# 16. CLOSURE
# =============================================================================

banner(
    "STAGE26-2-GIT COMPLETE"
)


print(
    "STAGE26-1 PARENT:"
)

print(
    " ",
    EXPECTED_PARENT,
)


print(
    "\nSTAGE26-2 WARM CPU COMMIT:"
)

print(
    " ",
    STAGE26_2_HEAD,
)


print(
    "\nREMOTE MAIN:"
)

print(
    " ",
    REMOTE_HEAD,
)


print(
    "\nCONDITION OUTCOMES:"
)

print(
    "  PASS                   : 73"
)

print(
    "  RESOURCE_LIMIT_OOM     : 2"
)

print(
    "  TIMEOUT_RESOURCE_LIMIT : 5"
)

print(
    "  TOTAL                  : 80"
)


print(
    "\nRAW TIMING OBSERVATIONS:"
)

print(
    "  8120"
)


print(
    "\nRAW JSONL SHA256:"
)

print(
    " ",
    EXPECTED_RAW_JSONL_SHA256,
)


print(
    "\nRAW CSV SHA256:"
)

print(
    " ",
    EXPECTED_RAW_CSV_SHA256,
)


print(
    "\nSUMMARY CSV SHA256:"
)

print(
    " ",
    EXPECTED_SUMMARY_CSV_SHA256,
)


print(
    "\nINTERRUPTION RECORD SHA256:"
)

print(
    " ",
    EXPECTED_INTERRUPTION_SHA256,
)


print(
    "\nRESUME RECORD SHA256:"
)

print(
    " ",
    EXPECTED_RESUME_SHA256,
)


print(
    "\nRECEIPT SHA256:"
)

print(
    " ",
    EXPECTED_RECEIPT_SHA256,
)


print(
    "\nPACKAGE MANIFEST SHA256:"
)

print(
    " ",
    EXPECTED_MANIFEST_SHA256,
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  COMMIT MATCH                   : PASS"
)

print(
    "  LS-REMOTE                      : PASS"
)

print(
    "  ALL MANIFEST-LISTED FILES      : PASS"
)

print(
    "  80 CONDITION RECEIPTS          : PASS"
)

print(
    "  RAW TIMINGS                    : PASS"
)

print(
    "  SUMMARY                        : PASS"
)

print(
    "  INTERRUPTION PROVENANCE        : PASS"
)

print(
    "  RESUME PROVENANCE              : PASS"
)

print(
    "  ORIGINAL PROTOCOL              : PASS"
)

print(
    "  ORIGINAL CPU PLAN              : PASS"
)

print(
    "  ORIGINAL PREFLIGHT             : PASS"
)

print(
    "  ORIGINAL COLD-START            : PASS"
)

print(
    "  REPOSITORY CLEAN               : PASS"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  STAGE26-2 CPU WARM PROFILING DURABLY FROZEN"
)

print(
    "  80 / 80 CONDITION OUTCOMES PRESERVED"
)

print(
    "  8120 RAW TIMING OBSERVATIONS PRESERVED"
)

print(
    "  2 OOM OUTCOMES PRESERVED"
)

print(
    "  5 TIMEOUT OUTCOMES PRESERVED"
)

print(
    "  OPERATOR INTERRUPTION TRANSPARENTLY PRESERVED"
)

print(
    "  NO COMPLETED CONDITION REMEASURED"
)

print(
    "  NO MEASUREMENT POLICY CHANGED"
)

print(
    "  MEMORY NOT YET PROFILED"
)

print(
    "  EXTRACTION NOT YET PROFILED"
)

print(
    "  END-TO-END NOT YET PROFILED"
)

print(
    "  PARETO NOT YET PERFORMED"
)

print(
    "  GPU NOT USED"
)

print(
    "  HOLDOUT NOT REOPENED"
)


print(
    "\nNEXT:"
)

print(
    "  STAGE26-3 — MEMORY + SERIALIZED / DEPLOYMENT PACKAGE FOOTPRINT"
)

print(
    "  GPU REMAINS OFF."
)


STAGE26-2-GIT :: LOCAL TOP-LEVEL HASH GATE
package manifest         PASS 7f4d01bb3fe685dca528a4a1c788bc5819b42a4e3080098fc3e923b2fb0e0f19
warm CPU receipt         PASS b4b2623eabde7dd6b9acc250357a9f1ad61fa342c1dd496dcb634d4bfaca4d15
raw JSONL                PASS 64e7961dc8a37c00ec08a34f77a6d50a8a5ec352ff1b75683360abc1b7147438
raw CSV                  PASS 78c58289ccfc4598966d6516201028f43c4b1d16d8bd0268a0cf58129d4179fa
summary CSV              PASS 75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232
interruption record      PASS 55e95ccc3426f52379931ff1126e4a6d8dfbf06ca75cd1c90dda18c078c6dcf8
resume record            PASS a28cac34a5dfd45db7a172ee30cbfe2efa91a8f75dfa127b46eb41f2dd42a2e3

STAGE26-2-GIT :: MANIFEST-COMPLETE LOCAL GATE
Manifest-listed files: 91
results/stage26_deployment_profiling/stage26_2_cpu_warm_inference/condition_receipts/condition_001.json   sha=PASS size=PASS
results/stage26_deployment_profiling/stage26_2_cpu_warm_inference/condition_receipts/conditio

In [21]:
# =============================================================================
# STAGE26-3A — MEMORY + DEPLOYMENT-PACKAGE IMPLEMENTATION FREEZE
#
# PARENT:
#   7ffdba3f4ca4ea5cc53097d62aaf27957009b9f6
#
# IMPORTANT:
#   THIS CELL PERFORMS ZERO MEMORY MEASUREMENTS.
#   THIS CELL PERFORMS ZERO MODEL INFERENCE.
#   THIS CELL DOES NOT LOAD ANY MODEL.
#
# PURPOSE
# -------
# Freeze the previously under-specified execution details of Stage26 memory
# profiling BEFORE observing any memory result.
#
# MEMORY SCOPE
# ------------
# Inherit the COMPLETE already-frozen CPU execution matrix:
#
#   8 targets
#   × 2 CPU modes
#   × 5 batch sizes
#   × 5 fresh-process memory repetitions
#   = 400 isolated memory repetitions
#
# No condition is selected or omitted using Stage26-2 latency/OOM results.
#
# FROZEN MEMORY SEMANTICS
# -----------------------
# baseline_rss:
#   after required framework imports + deterministic input preparation,
#   before model deserialization/construction
#
# loaded_rss:
#   after model construction/deserialization, before inference
#
# peak_rss:
#   maximum sampled process RSS during ONE untimed memory-only inference pass,
#   sampled every 5 ms
#
# delta_model_rss:
#   loaded_rss - baseline_rss
#
# delta_peak_rss:
#   peak_rss - baseline_rss
#
# ru_maxrss:
#   descriptive only
#
# max worker RAM:
#   85% of physical RAM; if crossed, retain RESOURCE_LIMIT_OOM
#
# PACKAGE FOOTPRINT
# -----------------
# serialized_model_size:
#   exact immutable model/checkpoint bytes
#
# deployment_package_size:
#   serialized artifact(s) + model-specific required preprocessing /
#   configuration / executable architecture representation.
#
# Generic Python/framework installation size is EXCLUDED.
#
# GPU:
#   OFF.
# =============================================================================

from __future__ import annotations

import os
import sys
import json
import csv
import hashlib
import shutil
import subprocess
import textwrap
from pathlib import Path
from datetime import datetime, timezone

import numpy as np


# =============================================================================
# 0. PATHS / FROZEN HASHES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

MEMORY_ROOT = (
    STAGE26_ROOT
    / "memory"
)

WORKERS_ROOT = (
    STAGE26_ROOT
    / "workers"
)

REPO_LOCK_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_3a_memory_implementation_lock"
)


MEMORY_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

WORKERS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


EXPECTED_HEAD = (
    "7ffdba3f4ca4ea5cc53097d62aaf27957009b9f6"
)

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

EXPECTED_CPU_PLAN_SHA256 = (
    "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363"
)

EXPECTED_STAGE26_2_RECEIPT_SHA256 = (
    "b4b2623eabde7dd6b9acc250357a9f1ad61fa342c1dd496dcb634d4bfaca4d15"
)


PROTOCOL = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_0_protocol_lock/"
      "measurement_protocol.json"
)

CPU_PLAN = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_0_protocol_lock/"
      "cpu_execution_plan.json"
)

STAGE26_2_RECEIPT = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_2_cpu_warm_inference/"
      "stage26_2_warm_cpu_receipt.json"
)


# =============================================================================
# 1. OUTPUTS
# =============================================================================

MEMORY_PLAN_PATH = (
    MEMORY_ROOT
    / "stage26_3_memory_execution_plan.json"
)

PACKAGE_MAP_PATH = (
    MEMORY_ROOT
    / "stage26_3_deployment_package_map.json"
)

IMPLEMENTATION_PATH = (
    MEMORY_ROOT
    / "stage26_3_memory_implementation.json"
)

WORKER_PATH = (
    WORKERS_ROOT
    / "stage26_memory_worker.py"
)

FREEZE_RECEIPT_PATH = (
    MEMORY_ROOT
    / "stage26_3a_memory_freeze_receipt.json"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def banner(text):
    print(
        "\n"
        + "=" * 112
    )

    print(text)

    print(
        "=" * 112
    )


def git(*args):

    p = subprocess.run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )


    os.replace(
        tmp,
        path,
    )


# =============================================================================
# 3. PARENT / IMMUTABILITY GATE
# =============================================================================

banner(
    "STAGE26-3A :: PARENT / IMMUTABILITY GATE"
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD :",
    EXPECTED_HEAD,
)

print(
    "Local HEAD    :",
    head,
)

print(
    "origin/main   :",
    remote,
)

print(
    "Repository clean:",
    status == "",
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected local HEAD before Stage26-3A."
    )


if remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main does not match anchored Stage26-2 commit."
    )


if status:

    raise RuntimeError(
        "Repository must be clean before Stage26-3A:\n"
        + status
    )


for name, path, expected in [
    (
        "measurement protocol",
        PROTOCOL,
        EXPECTED_PROTOCOL_SHA256,
    ),
    (
        "CPU execution plan",
        CPU_PLAN,
        EXPECTED_CPU_PLAN_SHA256,
    ),
    (
        "Stage26-2 warm receipt",
        STAGE26_2_RECEIPT,
        EXPECTED_STAGE26_2_RECEIPT_SHA256,
    ),
]:

    if not path.exists():

        raise FileNotFoundError(
            path
        )


    actual = sha256_file(
        path
    )


    print(
        f"{name:28s} "
        f"{'PASS' if actual == expected else 'FAIL'} "
        f"{actual}"
    )


    if actual != expected:

        raise RuntimeError(
            f"Frozen upstream artifact mismatch: {name}"
        )


# =============================================================================
# 4. READ AND ASSERT ORIGINAL MEMORY PROTOCOL
# =============================================================================

banner(
    "STAGE26-3A :: ORIGINAL MEMORY PROTOCOL"
)


protocol = json.loads(
    PROTOCOL.read_text(
        encoding="utf-8"
    )
)


memory_protocol = protocol[
    "memory_protocol"
]


print(
    json.dumps(
        memory_protocol,
        indent=2,
        sort_keys=True,
    )
)


assert (
    memory_protocol[
        "fresh_process_per_repetition"
    ]
    is True
)

assert (
    int(
        memory_protocol[
            "repetitions"
        ]
    )
    ==
    5
)

assert (
    int(
        memory_protocol[
            "rss_sampling_interval_ms"
        ]
    )
    ==
    5
)

assert (
    float(
        memory_protocol[
            "max_worker_ram_fraction"
        ]
    )
    ==
    0.85
)

assert (
    memory_protocol[
        "timing_and_memory_runs_separate"
    ]
    is True
)


print(
    "\n[PASS] Original Stage26 memory rules recovered exactly."
)


# =============================================================================
# 5. LOAD COMPLETE FROZEN 80-CONDITION CPU MATRIX
# =============================================================================

banner(
    "STAGE26-3A :: INHERIT COMPLETE CPU CONDITION MATRIX"
)


cpu_plan = json.loads(
    CPU_PLAN.read_text(
        encoding="utf-8"
    )
)


conditions = sorted(
    cpu_plan[
        "conditions"
    ],
    key=lambda row: int(
        row[
            "execution_order"
        ]
    ),
)


if len(
    conditions
) != 80:

    raise RuntimeError(
        "Expected exactly 80 frozen CPU conditions."
    )


expected_batches = {
    1,
    64,
    256,
    1024,
    8192,
}


if {
    int(
        row[
            "batch_size"
        ]
    )
    for row in conditions
} != expected_batches:

    raise RuntimeError(
        "Frozen CPU matrix does not contain expected five batch sizes."
    )


print(
    "Inherited frozen conditions:",
    len(
        conditions
    ),
)


# =============================================================================
# 6. FREEZE NON-SELECTIVE 400-REPETITION MEMORY PLAN
# =============================================================================

banner(
    "STAGE26-3A :: FREEZE 400-REPETITION MEMORY PLAN"
)


memory_repetitions = []


for condition in conditions:

    for repetition in range(
        1,
        6,
    ):

        memory_repetitions.append(
            {
                "source_cpu_execution_order":
                    int(
                        condition[
                            "execution_order"
                        ]
                    ),

                "condition_id":
                    condition[
                        "condition_id"
                    ],

                "target_id":
                    condition[
                        "target_id"
                    ],

                "target_role":
                    condition[
                        "target_role"
                    ],

                "comparison_group":
                    condition[
                        "comparison_group"
                    ],

                "hardware_mode":
                    condition[
                        "hardware_mode"
                    ],

                "thread_count":
                    int(
                        condition[
                            "thread_count"
                        ]
                    ),

                "affinity":
                    [
                        int(x)
                        for x in condition[
                            "affinity"
                        ]
                    ],

                "batch_size":
                    int(
                        condition[
                            "batch_size"
                        ]
                    ),

                "memory_repetition":
                    repetition,
            }
        )


if len(
    memory_repetitions
) != 400:

    raise RuntimeError(
        "Expected exactly 400 memory repetitions."
    )


# Deterministic randomization occurs BEFORE any memory observation.
rng = np.random.default_rng(
    26042
)


permutation = rng.permutation(
    400
)


ordered_memory_plan = []


for memory_execution_order, index in enumerate(
    permutation,
    start=1,
):

    row = dict(
        memory_repetitions[
            int(
                index
            )
        ]
    )

    row[
        "memory_execution_order"
    ] = memory_execution_order

    ordered_memory_plan.append(
        row
    )


memory_plan = {
    "schema":
        "stage26_3_memory_execution_plan_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-3",

    "frozen_before_first_memory_observation":
        True,

    "parent_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "source_cpu_execution_plan_sha256":
        EXPECTED_CPU_PLAN_SHA256,

    "selection_rule":
        (
            "INHERIT_ALL_80_ALREADY_FROZEN_CPU_CONDITIONS; "
            "NO BATCH/HARDWARE/TARGET SELECTION FROM STAGE26-2 RESULTS"
        ),

    "randomization_seed":
        26042,

    "condition_count":
        80,

    "repetitions_per_condition":
        5,

    "total_memory_repetitions":
        400,

    "memory_repetitions":
        ordered_memory_plan,
}


atomic_json(
    MEMORY_PLAN_PATH,
    memory_plan,
)


memory_plan_sha = sha256_file(
    MEMORY_PLAN_PATH
)


print(
    "Memory plan SHA256:"
)

print(
    " ",
    memory_plan_sha,
)


print(
    "\nFirst 15 memory repetitions:"
)


for row in ordered_memory_plan[
    :15
]:

    print(
        f"{row['memory_execution_order']:03d} "
        f"{row['target_id']:43s} "
        f"{row['hardware_mode']:21s} "
        f"B={row['batch_size']:5d} "
        f"rep={row['memory_repetition']}"
    )


# =============================================================================
# 7. FREEZE SERIALIZED + DEPLOYMENT PACKAGE DEFINITIONS
# =============================================================================

banner(
    "STAGE26-3A :: DEPLOYMENT PACKAGE DEFINITIONS"
)


# Exact Stage26 execution artifacts.
paths = {
    # Classical
    "xgb":
        "results/stage16_classical_benchmark_checkpoint/"
        "stage16_3_tuned_models/XGBOOST_tuned.joblib",

    "lgbm":
        "results/stage16_classical_benchmark_checkpoint/"
        "stage16_3_tuned_models/LIGHTGBM_tuned.joblib",

    "cat":
        "results/stage16_classical_benchmark_checkpoint/"
        "stage16_3_tuned_models/CATBOOST_tuned.joblib",

    # FT
    "ft_seed7":
        "results/stage15_transformer_checkpoint/"
        "stage15_4b_models/FT_BALANCED_seed_7_best_extended.pt",

    "ft_seed29":
        "results/stage15_transformer_checkpoint/"
        "stage15_4a_models/FT_BALANCED_seed_29_best.pt",

    "ft_seed101":
        "results/stage15_transformer_checkpoint/"
        "stage15_4a_models/FT_BALANCED_seed_101_best.pt",

    "ft_seed313":
        "results/stage15_transformer_checkpoint/"
        "stage15_4c_models/FT_BALANCED_seed_313_best.pt",

    "ft_seed997":
        "results/stage15_transformer_checkpoint/"
        "stage15_4c_models/FT_BALANCED_seed_997_best.pt",

    "ft_scaler":
        "results/stage15_transformer_checkpoint/"
        "stage15_2_standard_scaler.joblib",

    "ft_source":
        "results/stage15_transformer_checkpoint/"
        "ft_transformer_numeric.py",

    "ft_arch":
        "results/stage15_transformer_checkpoint/"
        "stage15_4c_frozen_architecture.json",

    # CNN
    "cnn_checkpoint":
        "results/stage20_1e_training/"
        "stage20_1e2_epoch10_model_state_dict.pt",

    "cnn_source":
        "scripts/stage20_masked_cnn.py",

    # ViT
    "vit_checkpoint":
        "results/stage21_architecture/"
        "stage21_2_epoch10_model_state_dict.pt",

    "vit_source":
        "scripts/stage21_masked_vit.py",

    # Shared packet representation
    "packet_encoder":
        "scripts/stage20_packet_image_encoder.py",
}


# -------------------------------------------------------------------------
# Serialized model artifacts:
# model/checkpoint bytes ONLY.
#
# Deployment package:
# serialized artifact(s) + exact required Stage26 executable architecture /
# preprocessing / representation/config artifacts.
# -------------------------------------------------------------------------

package_definitions = {
    "STAGE16_XGBOOST_TUNED": {
        "serialized_model_artifacts": [
            paths[
                "xgb"
            ],
        ],

        "deployment_package_artifacts": [
            paths[
                "xgb"
            ],
        ],
    },


    "STAGE16_LIGHTGBM_TUNED": {
        "serialized_model_artifacts": [
            paths[
                "lgbm"
            ],
        ],

        "deployment_package_artifacts": [
            paths[
                "lgbm"
            ],
        ],
    },


    "STAGE16_CATBOOST_TUNED": {
        "serialized_model_artifacts": [
            paths[
                "cat"
            ],
        ],

        "deployment_package_artifacts": [
            paths[
                "cat"
            ],
        ],
    },


    "ENS_LGBM_XGB_EQUAL": {
        "serialized_model_artifacts": [
            paths[
                "xgb"
            ],
            paths[
                "lgbm"
            ],
        ],

        "deployment_package_artifacts": [
            paths[
                "xgb"
            ],
            paths[
                "lgbm"
            ],
        ],
    },


    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE": {
        "serialized_model_artifacts": [
            paths[
                "ft_seed7"
            ],
        ],

        "deployment_package_artifacts": [
            paths[
                "ft_seed7"
            ],
            paths[
                "ft_scaler"
            ],
            paths[
                "ft_source"
            ],
            paths[
                "ft_arch"
            ],
        ],
    },


    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING": {
        "serialized_model_artifacts": [
            paths[
                "ft_seed7"
            ],
            paths[
                "ft_seed29"
            ],
            paths[
                "ft_seed101"
            ],
            paths[
                "ft_seed313"
            ],
            paths[
                "ft_seed997"
            ],
        ],

        "deployment_package_artifacts": [
            paths[
                "ft_seed7"
            ],
            paths[
                "ft_seed29"
            ],
            paths[
                "ft_seed101"
            ],
            paths[
                "ft_seed313"
            ],
            paths[
                "ft_seed997"
            ],
            paths[
                "ft_scaler"
            ],
            paths[
                "ft_source"
            ],
            paths[
                "ft_arch"
            ],
        ],
    },


    "STAGE20_MASKED_CNN_V1": {
        "serialized_model_artifacts": [
            paths[
                "cnn_checkpoint"
            ],
        ],

        "deployment_package_artifacts": [
            paths[
                "cnn_checkpoint"
            ],
            paths[
                "cnn_source"
            ],
            paths[
                "packet_encoder"
            ],
        ],
    },


    "STAGE21_MASKED_VIT_V1": {
        "serialized_model_artifacts": [
            paths[
                "vit_checkpoint"
            ],
        ],

        "deployment_package_artifacts": [
            paths[
                "vit_checkpoint"
            ],
            paths[
                "vit_source"
            ],
            paths[
                "packet_encoder"
            ],
        ],
    },
}


expected_targets = {
    row[
        "target_id"
    ]
    for row in conditions
}


if set(
    package_definitions
) != expected_targets:

    raise RuntimeError(
        "Deployment package target universe differs from frozen CPU target universe."
    )


# Resolve exact bytes + SHA values NOW, before memory measurements.
resolved_packages = {}


for target_id, definition in package_definitions.items():

    serialized_rows = []

    deployment_rows = []


    for relpath in definition[
        "serialized_model_artifacts"
    ]:

        path = (
            REPO
            / relpath
        )


        if not path.exists():

            raise FileNotFoundError(
                path
            )


        serialized_rows.append(
            {
                "repo_relative_path":
                    relpath,

                "size_bytes":
                    int(
                        path.stat().st_size
                    ),

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )


    for relpath in definition[
        "deployment_package_artifacts"
    ]:

        path = (
            REPO
            / relpath
        )


        if not path.exists():

            raise FileNotFoundError(
                path
            )


        deployment_rows.append(
            {
                "repo_relative_path":
                    relpath,

                "size_bytes":
                    int(
                        path.stat().st_size
                    ),

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )


    serialized_size = int(
        sum(
            row[
                "size_bytes"
            ]
            for row in serialized_rows
        )
    )


    deployment_size = int(
        sum(
            row[
                "size_bytes"
            ]
            for row in deployment_rows
        )
    )


    resolved_packages[
        target_id
    ] = {
        "serialized_model_artifacts":
            serialized_rows,

        "deployment_package_artifacts":
            deployment_rows,

        "serialized_model_size_bytes":
            serialized_size,

        "deployment_package_size_bytes":
            deployment_size,

        "generic_framework_installation_excluded":
            True,
    }


package_map = {
    "schema":
        "stage26_3_deployment_package_map_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-3",

    "frozen_before_first_memory_observation":
        True,

    "definition": {
        "serialized_model_size":
            (
                "Sum of immutable Stage26 model/checkpoint artifacts "
                "for the actual deployment unit."
            ),

        "deployment_package_size":
            (
                "Serialized artifact(s) plus exact model-specific "
                "preprocessing/configuration/executable architecture/"
                "representation artifacts used by Stage26 execution."
            ),

        "generic_python_or_framework_install_size_included":
            False,
    },

    "targets":
        resolved_packages,
}


atomic_json(
    PACKAGE_MAP_PATH,
    package_map,
)


package_map_sha = sha256_file(
    PACKAGE_MAP_PATH
)


print(
    "Deployment package map SHA256:"
)

print(
    " ",
    package_map_sha,
)


print(
    "\nFrozen package sizes:"
)


for target_id in sorted(
    resolved_packages
):

    row = resolved_packages[
        target_id
    ]


    print(
        f"{target_id:43s} "
        f"serialized="
        f"{row['serialized_model_size_bytes'] / 1024**2:9.3f} MiB "
        f"deployment="
        f"{row['deployment_package_size_bytes'] / 1024**2:9.3f} MiB"
    )


# =============================================================================
# 8. FREEZE MEMORY WORKER SOURCE — DO NOT EXECUTE IT
# =============================================================================

banner(
    "STAGE26-3A :: FREEZE MEMORY WORKER SOURCE"
)


worker_source = r'''
from __future__ import annotations

import os
import sys
import gc
import json
import time
import queue
import signal
import resource
import threading
import traceback
import importlib.util
from pathlib import Path

import psutil


RSS_SAMPLE_INTERVAL_SECONDS = 0.005


def atomic_json(path, obj):

    path = Path(path)

    tmp = Path(
        str(path)
        +
        ".tmp"
    )

    raw = (
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
        )
        +
        "\n"
    ).encode(
        "utf-8"
    )


    with tmp.open(
        "wb"
    ) as f:

        f.write(
            raw
        )

        f.flush()

        os.fsync(
            f.fileno()
        )


    os.replace(
        tmp,
        path,
    )


def import_path(name, path):

    spec = importlib.util.spec_from_file_location(
        name,
        str(path),
    )

    if (
        spec is None
        or
        spec.loader is None
    ):

        raise RuntimeError(
            f"Unable to import {path}"
        )


    module = importlib.util.module_from_spec(
        spec
    )

    spec.loader.exec_module(
        module
    )

    return module


def extract_torch_state(obj):

    import torch

    if not isinstance(
        obj,
        dict,
    ):

        raise TypeError(
            f"Unsupported checkpoint root type: {type(obj)}"
        )


    if (
        obj
        and
        all(
            torch.is_tensor(v)
            for v in obj.values()
        )
    ):

        return obj


    for key in [
        "model_state_dict",
        "state_dict",
        "model_state",
        "network_state_dict",
        "model",
    ]:

        candidate = obj.get(
            key
        )


        if (
            isinstance(
                candidate,
                dict,
            )
            and
            candidate
            and
            all(
                torch.is_tensor(v)
                for v in candidate.values()
            )
        ):

            return candidate


    raise RuntimeError(
        "Unable to identify torch state_dict."
    )


def torch_load_state(path):

    import torch

    try:

        obj = torch.load(
            str(path),
            map_location="cpu",
            weights_only=True,
        )


    except TypeError:

        obj = torch.load(
            str(path),
            map_location="cpu",
        )


    return extract_torch_state(
        obj
    )


def configure_torch_threads(
    thread_count,
):

    import torch

    torch.set_num_threads(
        int(
            thread_count
        )
    )


    try:

        torch.set_num_interop_threads(
            1
        )


    except RuntimeError:

        pass


def make_ipv4_packet(
    rng,
    protocol,
    length,
):

    import numpy as np

    packet = bytearray(
        rng.integers(
            0,
            256,
            size=int(
                length
            ),
            dtype=np.uint8,
        ).tobytes()
    )

    packet[
        0
    ] = 0x45

    packet[
        2:4
    ] = int(
        length
    ).to_bytes(
        2,
        "big",
    )

    packet[
        6
    ] = (
        packet[
            6
        ]
        &
        0xE0
    )

    packet[
        7
    ] = 0

    packet[
        9
    ] = int(
        protocol
    )

    return bytes(
        packet
    )


def build_group_a_input(
    repo,
    batch_size,
    seed,
):

    import joblib
    import numpy as np

    scaler = joblib.load(
        repo
        / "results/stage15_transformer_checkpoint/"
          "stage15_2_standard_scaler.joblib"
    )


    derived_seed = (
        int(seed)
        +
        int(batch_size)
    )

    rng = np.random.default_rng(
        derived_seed
    )


    Z = rng.normal(
        0.0,
        1.0,
        size=(
            batch_size,
            70,
        ),
    ).astype(
        np.float32
    )


    mean = np.asarray(
        scaler.mean_,
        dtype=np.float32,
    )

    scale = np.asarray(
        scaler.scale_,
        dtype=np.float32,
    )


    X_raw = (
        mean[
            None,
            :
        ]
        +
        Z
        *
        scale[
            None,
            :
        ]
    ).astype(
        np.float32,
        copy=False,
    )


    return (
        Z,
        X_raw,
    )


def build_packet_input(
    repo,
    batch_size,
    seed,
    need_scaled_float,
):

    import numpy as np

    encoder = import_path(
        "stage26_memory_encoder",
        repo
        / "scripts/stage20_packet_image_encoder.py",
    )


    derived_seed = (
        int(seed)
        +
        1_000_000
        +
        int(batch_size)
    )

    rng = np.random.default_rng(
        derived_seed
    )


    images = np.zeros(
        (
            batch_size,
            1,
            64,
            256,
        ),
        dtype=np.uint8,
    )

    masks = np.zeros(
        (
            batch_size,
            1,
            64,
            256,
        ),
        dtype=np.bool_,
    )


    for i in range(
        batch_size
    ):

        packet_count = int(
            rng.integers(
                1,
                65,
            )
        )

        packets = []


        for _ in range(
            packet_count
        ):

            protocol = (
                6
                if int(
                    rng.integers(
                        0,
                        2,
                    )
                ) == 0
                else 17
            )


            length = int(
                rng.integers(
                    40,
                    257,
                )
            )


            packets.append(
                make_ipv4_packet(
                    rng,
                    protocol,
                    length,
                )
            )


        image, mask = encoder.encode_flow(
            packets
        )


        images[
            i,
            0
        ] = image

        masks[
            i,
            0
        ] = mask


    if need_scaled_float:

        images = (
            images.astype(
                np.float32
            )
            /
            np.float32(
                255.0
            )
        )


    return (
        images,
        masks,
    )


def build_ft_model(
    ft_module,
    architecture_record,
    checkpoint,
):

    arch = architecture_record[
        "architecture"
    ]


    model = ft_module.NumericFTTransformer(
        n_features=int(
            architecture_record[
                "input_predictor_count"
            ]
        ),
        d_token=int(
            arch[
                "d_token"
            ]
        ),
        n_heads=int(
            arch[
                "n_heads"
            ]
        ),
        n_layers=int(
            arch[
                "n_layers"
            ]
        ),
        d_ff=int(
            arch[
                "d_ff"
            ]
        ),
        dropout=float(
            arch[
                "dropout"
            ]
        ),
    )


    model.load_state_dict(
        torch_load_state(
            checkpoint
        ),
        strict=True,
    )


    model.eval()

    return model


def start_rss_sampler(
    process,
    stop_event,
    samples,
):

    while not stop_event.is_set():

        try:

            rss = int(
                process.memory_info().rss
            )

            samples.append(
                rss
            )


        except psutil.NoSuchProcess:

            break


        stop_event.wait(
            RSS_SAMPLE_INTERVAL_SECONDS
        )


def main():

    config = json.loads(
        Path(
            sys.argv[
                1
            ]
        ).read_text(
            encoding="utf-8"
        )
    )


    result_path = Path(
        config[
            "result_path"
        ]
    )


    repo = Path(
        config[
            "repo"
        ]
    )

    target = config[
        "target_id"
    ]

    batch_size = int(
        config[
            "batch_size"
        ]
    )

    thread_count = int(
        config[
            "thread_count"
        ]
    )

    affinity = [
        int(x)
        for x in config[
            "affinity"
        ]
    ]

    seed = int(
        config[
            "measurement_seed"
        ]
    )


    if hasattr(
        os,
        "sched_setaffinity",
    ):

        os.sched_setaffinity(
            0,
            set(
                affinity
            ),
        )


    process = psutil.Process(
        os.getpid()
    )


    # -------------------------------------------------------------------------
    # Required imports + deterministic input preparation.
    # -------------------------------------------------------------------------

    if target in {
        "STAGE16_XGBOOST_TUNED",
        "STAGE16_LIGHTGBM_TUNED",
        "STAGE16_CATBOOST_TUNED",
        "ENS_LGBM_XGB_EQUAL",
    }:

        import numpy as np
        import joblib


        Z, X_raw = build_group_a_input(
            repo,
            batch_size,
            seed,
        )


    elif target in {
        "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
        "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    }:

        import numpy as np
        import torch


        configure_torch_threads(
            thread_count
        )


        Z, _ = build_group_a_input(
            repo,
            batch_size,
            seed,
        )


        x = torch.from_numpy(
            Z
        )


        ft_module = import_path(
            "stage26_memory_ft",
            repo
            / "results/stage15_transformer_checkpoint/"
              "ft_transformer_numeric.py",
        )


    elif target == "STAGE20_MASKED_CNN_V1":

        import numpy as np
        import torch


        configure_torch_threads(
            thread_count
        )


        image_np, mask_np = build_packet_input(
            repo,
            batch_size,
            seed,
            False,
        )


        image = torch.from_numpy(
            image_np
        )

        mask = torch.from_numpy(
            mask_np
        )


        cnn_module = import_path(
            "stage26_memory_cnn",
            repo
            / "scripts/stage20_masked_cnn.py",
        )


    elif target == "STAGE21_MASKED_VIT_V1":

        import numpy as np
        import torch


        configure_torch_threads(
            thread_count
        )


        image_np, mask_np = build_packet_input(
            repo,
            batch_size,
            seed,
            True,
        )


        image = torch.from_numpy(
            image_np
        )

        mask = torch.from_numpy(
            mask_np
        )


        vit_module = import_path(
            "stage26_memory_vit",
            repo
            / "scripts/stage21_masked_vit.py",
        )


    else:

        raise RuntimeError(
            f"Unknown target: {target}"
        )


    baseline_rss = int(
        process.memory_info().rss
    )


    # -------------------------------------------------------------------------
    # Model load/construction.
    # -------------------------------------------------------------------------

    if target == "STAGE16_XGBOOST_TUNED":

        import xgboost


        model = joblib.load(
            repo
            / "results/stage16_classical_benchmark_checkpoint/"
              "stage16_3_tuned_models/XGBOOST_tuned.joblib"
        )


        model.set_params(
            n_jobs=thread_count
        )


        def infer():

            return model.predict_proba(
                X_raw
            )[
                :,
                1
            ]


    elif target == "STAGE16_LIGHTGBM_TUNED":

        import lightgbm


        model = joblib.load(
            repo
            / "results/stage16_classical_benchmark_checkpoint/"
              "stage16_3_tuned_models/LIGHTGBM_tuned.joblib"
        )


        model.set_params(
            n_jobs=thread_count
        )


        def infer():

            return model.predict_proba(
                X_raw
            )[
                :,
                1
            ]


    elif target == "STAGE16_CATBOOST_TUNED":

        import catboost


        model = joblib.load(
            repo
            / "results/stage16_classical_benchmark_checkpoint/"
              "stage16_3_tuned_models/CATBOOST_tuned.joblib"
        )


        def infer():

            return model.predict_proba(
                X_raw,
                thread_count=thread_count,
            )[
                :,
                1
            ]


    elif target == "ENS_LGBM_XGB_EQUAL":

        import xgboost
        import lightgbm


        xgb_model = joblib.load(
            repo
            / "results/stage16_classical_benchmark_checkpoint/"
              "stage16_3_tuned_models/XGBOOST_tuned.joblib"
        )

        lgb_model = joblib.load(
            repo
            / "results/stage16_classical_benchmark_checkpoint/"
              "stage16_3_tuned_models/LIGHTGBM_tuned.joblib"
        )


        xgb_model.set_params(
            n_jobs=thread_count
        )

        lgb_model.set_params(
            n_jobs=thread_count
        )


        def infer():

            a = xgb_model.predict_proba(
                X_raw
            )[
                :,
                1
            ]

            b = lgb_model.predict_proba(
                X_raw
            )[
                :,
                1
            ]

            return (
                np.asarray(
                    a,
                    dtype=np.float64,
                )
                +
                np.asarray(
                    b,
                    dtype=np.float64,
                )
            ) / 2.0


    elif target in {
        "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
        "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    }:

        architecture_record = json.loads(
            (
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4c_frozen_architecture.json"
            ).read_text(
                encoding="utf-8"
            )
        )


        checkpoint_paths = {
            7:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4b_models/"
                  "FT_BALANCED_seed_7_best_extended.pt",

            29:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4a_models/"
                  "FT_BALANCED_seed_29_best.pt",

            101:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4a_models/"
                  "FT_BALANCED_seed_101_best.pt",

            313:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4c_models/"
                  "FT_BALANCED_seed_313_best.pt",

            997:
                repo
                / "results/stage15_transformer_checkpoint/"
                  "stage15_4c_models/"
                  "FT_BALANCED_seed_997_best.pt",
        }


        checkpoint_seeds = (
            [
                7
            ]
            if target
            ==
            "FT_BALANCED_SINGLE_RESOURCE_REFERENCE"
            else
            [
                7,
                29,
                101,
                313,
                997,
            ]
        )


        models = [
            build_ft_model(
                ft_module,
                architecture_record,
                checkpoint_paths[
                    checkpoint_seed
                ],
            )
            for checkpoint_seed
            in checkpoint_seeds
        ]


        def infer():

            outputs = []


            with torch.inference_mode():

                for model in models:

                    outputs.append(
                        torch.sigmoid(
                            model(
                                x
                            )
                        )
                        .detach()
                        .cpu()
                        .numpy()
                    )


            return np.mean(
                np.stack(
                    outputs,
                    axis=0,
                ),
                axis=0,
            )


    elif target == "STAGE20_MASKED_CNN_V1":

        model = (
            cnn_module.Stage20MaskedCNNv1()
        )


        model.load_state_dict(
            torch_load_state(
                repo
                / "results/stage20_1e_training/"
                  "stage20_1e2_epoch10_model_state_dict.pt"
            ),
            strict=True,
        )


        model.eval()


        def infer():

            with torch.inference_mode():

                return (
                    torch.sigmoid(
                        model(
                            image,
                            mask,
                        )
                    )
                    .detach()
                    .cpu()
                    .numpy()
                )


    elif target == "STAGE21_MASKED_VIT_V1":

        model = (
            vit_module.Stage21MaskedViTv1()
        )


        model.load_state_dict(
            torch_load_state(
                repo
                / "results/stage21_architecture/"
                  "stage21_2_epoch10_model_state_dict.pt"
            ),
            strict=True,
        )


        model.eval()


        def infer():

            with torch.inference_mode():

                return (
                    torch.sigmoid(
                        model(
                            image,
                            mask,
                        )
                    )
                    .detach()
                    .cpu()
                    .numpy()
                )


    loaded_rss = int(
        process.memory_info().rss
    )


    # -------------------------------------------------------------------------
    # 5 ms sampled memory-only inference.
    # Exactly ONE inference pass.
    # Timing is intentionally not measured.
    # -------------------------------------------------------------------------

    samples = [
        loaded_rss
    ]

    stop_event = threading.Event()


    sampler = threading.Thread(
        target=start_rss_sampler,
        args=(
            process,
            stop_event,
            samples,
        ),
        daemon=True,
    )


    sampler.start()


    try:

        output = infer()


    finally:

        # Explicit post-inference RSS sample.
        try:

            samples.append(
                int(
                    process.memory_info().rss
                )
            )


        except Exception:

            pass


        stop_event.set()

        sampler.join(
            timeout=1.0
        )


    # Ensure output materialized.
    import numpy as np


    output = np.asarray(
        output
    ).reshape(
        -1
    )


    if output.shape != (
        batch_size,
    ):

        raise RuntimeError(
            f"Unexpected output shape: {output.shape}"
        )


    if not np.isfinite(
        output
    ).all():

        raise RuntimeError(
            "Non-finite model output."
        )


    peak_rss = int(
        max(
            samples
        )
    )


    ru = resource.getrusage(
        resource.RUSAGE_SELF
    )


    # Linux ru_maxrss is KiB.
    ru_maxrss_bytes = int(
        ru.ru_maxrss
        *
        1024
    )


    result = {
        "schema":
            "stage26_3_memory_observation_v1",

        "status":
            "PASS",

        "memory_execution_order":
            int(
                config[
                    "memory_execution_order"
                ]
            ),

        "source_cpu_execution_order":
            int(
                config[
                    "source_cpu_execution_order"
                ]
            ),

        "condition_id":
            config[
                "condition_id"
            ],

        "target_id":
            target,

        "hardware_mode":
            config[
                "hardware_mode"
            ],

        "thread_count":
            thread_count,

        "affinity_requested":
            affinity,

        "batch_size":
            batch_size,

        "memory_repetition":
            int(
                config[
                    "memory_repetition"
                ]
            ),

        "rss_sampling_interval_ms":
            5,

        "rss_sample_count":
            len(
                samples
            ),

        "baseline_rss_bytes":
            baseline_rss,

        "loaded_rss_bytes":
            loaded_rss,

        "peak_rss_bytes":
            peak_rss,

        "delta_model_rss_bytes":
            int(
                loaded_rss
                -
                baseline_rss
            ),

        "delta_peak_rss_bytes":
            int(
                peak_rss
                -
                baseline_rss
            ),

        "ru_maxrss_bytes_descriptive":
            ru_maxrss_bytes,

        "timing_performed":
            False,

        "inference_passes":
            1,

        "holdout_accessed":
            False,

        "gpu_used":
            False,
    }


    atomic_json(
        result_path,
        result,
    )


if __name__ == "__main__":

    main()
'''


WORKER_PATH.write_text(
    textwrap.dedent(
        worker_source
    ).lstrip(),
    encoding="utf-8",
)


worker_sha = sha256_file(
    WORKER_PATH
)


print(
    "Memory worker SHA256:"
)

print(
    " ",
    worker_sha,
)


# Static audit: timing APIs must not appear in worker.
worker_text = WORKER_PATH.read_text(
    encoding="utf-8"
)


for forbidden in [
    "perf_counter",
    "monotonic(",
    "process_time(",
    "time.time(",
]:

    if forbidden in worker_text:

        raise RuntimeError(
            "Timing API unexpectedly found in memory worker: "
            +
            forbidden
        )


print(
    "[PASS] Memory worker contains no performance timing API."
)


# =============================================================================
# 9. FREEZE IMPLEMENTATION RECORD
# =============================================================================

banner(
    "STAGE26-3A :: IMPLEMENTATION RECORD"
)


implementation = {
    "schema":
        "stage26_3_memory_implementation_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-3",

    "status":
        "FROZEN_BEFORE_FIRST_MEMORY_OBSERVATION",

    "frozen_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "source_cpu_execution_plan_sha256":
        EXPECTED_CPU_PLAN_SHA256,

    "stage26_2_receipt_sha256":
        EXPECTED_STAGE26_2_RECEIPT_SHA256,

    "memory_execution_plan_sha256":
        memory_plan_sha,

    "deployment_package_map_sha256":
        package_map_sha,

    "memory_worker_sha256":
        worker_sha,

    "memory_scope": {
        "target_count":
            8,

        "cpu_modes":
            2,

        "batch_sizes":
            [
                1,
                64,
                256,
                1024,
                8192,
            ],

        "source_condition_count":
            80,

        "fresh_process_repetitions_per_condition":
            5,

        "total_fresh_process_repetitions":
            400,

        "selection_uses_stage26_2_performance_results":
            False,
    },

    "memory_boundaries": {
        "baseline_rss":
            (
                "After required framework imports and deterministic input "
                "preparation, before model load."
            ),

        "loaded_rss":
            (
                "After model construction/deserialization and before inference."
            ),

        "peak_rss":
            (
                "Maximum RSS sampled every 5 ms during exactly one "
                "untimed memory-only inference pass; loaded and immediate "
                "post-inference RSS are also included in sample set."
            ),

        "delta_model_rss":
            "loaded_rss - baseline_rss",

        "delta_peak_rss":
            "peak_rss - baseline_rss",

        "ru_maxrss":
            "Linux ru_maxrss converted from KiB to bytes; descriptive only.",
    },

    "rss_sampling_interval_ms":
        5,

    "memory_inference_passes_per_repetition":
        1,

    "max_worker_ram_fraction":
        0.85,

    "resource_limit_policy":
        (
            "Parent execution controller must terminate and retain "
            "RESOURCE_LIMIT_OOM if worker RSS exceeds 85% of physical RAM. "
            "No batch replacement."
        ),

    "memory_and_timing_separate":
        True,

    "serialized_package_map_frozen_before_memory":
        True,

    "generic_framework_installation_size_excluded":
        True,

    "gpu_used":
        False,

    "holdout_accessed":
        False,

    "memory_observations_at_freeze":
        0,
}


atomic_json(
    IMPLEMENTATION_PATH,
    implementation,
)


implementation_sha = sha256_file(
    IMPLEMENTATION_PATH
)


print(
    "Implementation SHA256:"
)

print(
    " ",
    implementation_sha,
)


# =============================================================================
# 10. FREEZE RECEIPT
# =============================================================================

freeze_receipt = {
    "schema":
        "stage26_3a_memory_freeze_receipt_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-3A",

    "status":
        "PASS",

    "parent_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "cpu_execution_plan_sha256":
        EXPECTED_CPU_PLAN_SHA256,

    "stage26_2_receipt_sha256":
        EXPECTED_STAGE26_2_RECEIPT_SHA256,

    "memory_execution_plan_sha256":
        memory_plan_sha,

    "deployment_package_map_sha256":
        package_map_sha,

    "memory_worker_sha256":
        worker_sha,

    "implementation_sha256":
        implementation_sha,

    "source_cpu_condition_count":
        80,

    "memory_repetitions_per_condition":
        5,

    "planned_memory_observation_count":
        400,

    "memory_observations_performed":
        0,

    "model_deserialization_performed":
        False,

    "inference_performed":
        False,

    "timing_performed":
        False,

    "gpu_used":
        False,

    "holdout_reopened":
        False,

    "next_action":
        (
            "GIT_ANCHOR_STAGE26_3A_IMPLEMENTATION_BEFORE_FIRST_MEMORY_RUN"
        ),
}


atomic_json(
    FREEZE_RECEIPT_PATH,
    freeze_receipt,
)


freeze_receipt_sha = sha256_file(
    FREEZE_RECEIPT_PATH
)


print(
    "Freeze receipt SHA256:"
)

print(
    " ",
    freeze_receipt_sha,
)


# =============================================================================
# 11. BUILD DURABLE PRE-MEASUREMENT LOCK PACKAGE
# =============================================================================

banner(
    "STAGE26-3A :: BUILD PRE-MEASUREMENT LOCK PACKAGE"
)


if REPO_LOCK_DIR.exists():

    raise RuntimeError(
        "Stage26-3A repository lock package already exists."
    )


REPO_LOCK_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


for source in [
    MEMORY_PLAN_PATH,
    PACKAGE_MAP_PATH,
    IMPLEMENTATION_PATH,
    WORKER_PATH,
    FREEZE_RECEIPT_PATH,
]:

    shutil.copy2(
        source,
        REPO_LOCK_DIR
        / source.name,
    )


PACKAGE_MANIFEST = (
    REPO_LOCK_DIR
    / "stage26_3a_memory_lock_package_manifest.json"
)


manifest_files = []


for path in sorted(
    REPO_LOCK_DIR.iterdir()
):

    if (
        path.is_file()
        and
        path != PACKAGE_MANIFEST
    ):

        manifest_files.append(
            {
                "path":
                    str(
                        path.relative_to(
                            REPO
                        )
                    ),

                "size_bytes":
                    int(
                        path.stat().st_size
                    ),

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )


atomic_json(
    PACKAGE_MANIFEST,
    {
        "schema":
            "stage26_3a_memory_lock_package_manifest_v1",

        "stage":
            26,

        "checkpoint":
            "STAGE26-3A",

        "status":
            "READY_FOR_GIT_ANCHOR",

        "parent_commit":
            EXPECTED_HEAD,

        "measurement_protocol_sha256":
            EXPECTED_PROTOCOL_SHA256,

        "freeze_receipt_sha256":
            freeze_receipt_sha,

        "file_count_excluding_manifest":
            len(
                manifest_files
            ),

        "files":
            manifest_files,

        "scientific_boundary": {
            "memory_plan_frozen":
                True,

            "deployment_package_map_frozen":
                True,

            "memory_worker_frozen":
                True,

            "memory_observations":
                0,

            "model_deserialization_performed":
                False,

            "inference_performed":
                False,

            "timing_performed":
                False,

            "gpu_used":
                False,

            "holdout_reopened":
                False,
        },
    },
)


manifest_sha = sha256_file(
    PACKAGE_MANIFEST
)


# =============================================================================
# 12. FINAL AUDIT
# =============================================================================

banner(
    "STAGE26-3A FINAL AUDIT"
)


repo_status_after = git(
    "status",
    "--porcelain",
)


print(
    "Repository status:"
)

print(
    repo_status_after
    if repo_status_after
    else "<unexpected clean>"
)


if not repo_status_after:

    raise RuntimeError(
        "Expected new Stage26-3A lock package."
    )


unexpected = []


for line in repo_status_after.splitlines():

    relpath = line[
        3:
    ]


    if not relpath.startswith(
        "results/stage26_deployment_profiling/"
        "stage26_3a_memory_implementation_lock/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository modifications outside Stage26-3A:\n"
        +
        "\n".join(
            unexpected
        )
    )


# Reassert zero observations.
receipt_check = json.loads(
    FREEZE_RECEIPT_PATH.read_text(
        encoding="utf-8"
    )
)


if receipt_check[
    "memory_observations_performed"
] != 0:

    raise RuntimeError(
        "Memory observations unexpectedly exist before implementation anchor."
    )


banner(
    "STAGE26-3A MEMORY IMPLEMENTATION FREEZE COMPLETE"
)


print(
    "PARENT:"
)

print(
    " ",
    EXPECTED_HEAD,
)


print(
    "\nMEMORY EXECUTION PLAN SHA256:"
)

print(
    " ",
    memory_plan_sha,
)


print(
    "\nDEPLOYMENT PACKAGE MAP SHA256:"
)

print(
    " ",
    package_map_sha,
)


print(
    "\nMEMORY WORKER SHA256:"
)

print(
    " ",
    worker_sha,
)


print(
    "\nIMPLEMENTATION SHA256:"
)

print(
    " ",
    implementation_sha,
)


print(
    "\nFREEZE RECEIPT SHA256:"
)

print(
    " ",
    freeze_receipt_sha,
)


print(
    "\nPACKAGE MANIFEST SHA256:"
)

print(
    " ",
    manifest_sha,
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  80 CPU CONDITIONS INHERITED WITHOUT PERFORMANCE-BASED SELECTION"
)

print(
    "  5 FRESH MEMORY REPETITIONS PER CONDITION"
)

print(
    "  400 MEMORY REPETITIONS PREDECLARED"
)

print(
    "  5 ms RSS SAMPLING FROZEN"
)

print(
    "  BASELINE / LOADED / PEAK RSS BOUNDARIES FROZEN"
)

print(
    "  EXACTLY ONE UNTIMERED INFERENCE PASS PER MEMORY REP FROZEN"
)

print(
    "  85% WORKER-RAM LIMIT FROZEN"
)

print(
    "  SERIALIZED MODEL FOOTPRINT DEFINITIONS FROZEN"
)

print(
    "  DEPLOYMENT PACKAGE FOOTPRINT DEFINITIONS FROZEN"
)

print(
    "  MEMORY OBSERVATIONS = 0"
)

print(
    "  INFERENCE PERFORMED = FALSE"
)

print(
    "  PERFORMANCE TIMING PERFORMED = FALSE"
)

print(
    "  GPU NOT USED"
)

print(
    "  HOLDOUT NOT REOPENED"
)


print(
    "\nNEXT:"
)

print(
    "  STAGE26-3A-GIT — anchor this memory implementation BEFORE"
)

print(
    "  executing the first memory observation."
)


STAGE26-3A :: PARENT / IMMUTABILITY GATE
Expected HEAD : 7ffdba3f4ca4ea5cc53097d62aaf27957009b9f6
Local HEAD    : 7ffdba3f4ca4ea5cc53097d62aaf27957009b9f6
origin/main   : 7ffdba3f4ca4ea5cc53097d62aaf27957009b9f6
Repository clean: True
measurement protocol         PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
CPU execution plan           PASS b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363
Stage26-2 warm receipt       PASS b4b2623eabde7dd6b9acc250357a9f1ad61fa342c1dd496dcb634d4bfaca4d15

STAGE26-3A :: ORIGINAL MEMORY PROTOCOL
{
  "cpu_metrics": {
    "baseline_rss": "RSS after required framework imports and input preparation, before model load.",
    "delta_model_rss": "loaded_rss - baseline_rss",
    "delta_peak_rss": "peak_rss - baseline_rss",
    "loaded_rss": "RSS after model load and before inference.",
    "peak_rss": "Maximum sampled process RSS during memory-only inference run.",
    "ru_maxrss": "Recorded descriptively inside the isolat

In [22]:
# =============================================================================
# STAGE26-3A-GIT — ANCHOR MEMORY IMPLEMENTATION BEFORE FIRST OBSERVATION
#
# PARENT:
#   7ffdba3f4ca4ea5cc53097d62aaf27957009b9f6
#
# EXPECTED LOCKS:
#   memory plan:
#     018584b8f8c94d6afa5fcf3576c261bae2b0f694c8b748bf2e0421234e17ea13
#
#   package map:
#     09e0d093a8ca83f4d68e9affd675af4336b982a6af14e5034fc2e5ffd4bc547c
#
#   memory worker:
#     2ba87c88244b8b3d8b0265d44b29204dc87e321e5aedc35dc5ac536de5fa8fd7
#
#   implementation:
#     9e8439d41924f5bdd13dc4db0a8e997fb9a6be1883a38a9ff4028c9f785452dc
#
#   freeze receipt:
#     465cc0c89f09c42f0d33b4dc1f8e759983fccd2897cef0207c85e777cdee3a19
#
#   package manifest:
#     de1b2731a276d4628360478f704ac16feabcad97925f3f1079657f4aced24eb1
#
# THIS CELL:
#   - performs NO model loading
#   - performs NO inference
#   - performs NO memory measurement
#   - performs NO timing
#   - uses NO GPU
#   - accesses NO holdout
#
# IDEMPOTENT:
#   Safe if accidentally executed twice after a successful commit.
# =============================================================================

from __future__ import annotations

import os
import json
import stat
import hashlib
import tempfile
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "7ffdba3f4ca4ea5cc53097d62aaf27957009b9f6"
)

COMMIT_MESSAGE = (
    "stage26: freeze memory profiling implementation"
)

LOCK_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_3a_memory_implementation_lock"
)

PLAN_REL = (
    LOCK_REL
    + "/stage26_3_memory_execution_plan.json"
)

PACKAGE_MAP_REL = (
    LOCK_REL
    + "/stage26_3_deployment_package_map.json"
)

WORKER_REL = (
    LOCK_REL
    + "/stage26_memory_worker.py"
)

IMPLEMENTATION_REL = (
    LOCK_REL
    + "/stage26_3_memory_implementation.json"
)

FREEZE_RECEIPT_REL = (
    LOCK_REL
    + "/stage26_3a_memory_freeze_receipt.json"
)

MANIFEST_REL = (
    LOCK_REL
    + "/stage26_3a_memory_lock_package_manifest.json"
)


EXPECTED_PLAN_SHA256 = (
    "018584b8f8c94d6afa5fcf3576c261bae2b0f694c8b748bf2e0421234e17ea13"
)

EXPECTED_PACKAGE_MAP_SHA256 = (
    "09e0d093a8ca83f4d68e9affd675af4336b982a6af14e5034fc2e5ffd4bc547c"
)

EXPECTED_WORKER_SHA256 = (
    "2ba87c88244b8b3d8b0265d44b29204dc87e321e5aedc35dc5ac536de5fa8fd7"
)

EXPECTED_IMPLEMENTATION_SHA256 = (
    "9e8439d41924f5bdd13dc4db0a8e997fb9a6be1883a38a9ff4028c9f785452dc"
)

EXPECTED_FREEZE_RECEIPT_SHA256 = (
    "465cc0c89f09c42f0d33b4dc1f8e759983fccd2897cef0207c85e777cdee3a19"
)

EXPECTED_MANIFEST_SHA256 = (
    "de1b2731a276d4628360478f704ac16feabcad97925f3f1079657f4aced24eb1"
)


# Upstream immutable anchors.
PROTOCOL_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/"
    "measurement_protocol.json"
)

CPU_PLAN_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/"
    "cpu_execution_plan.json"
)

STAGE26_2_RECEIPT_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_2_cpu_warm_inference/"
    "stage26_2_warm_cpu_receipt.json"
)


EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

EXPECTED_CPU_PLAN_SHA256 = (
    "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363"
)

EXPECTED_STAGE26_2_RECEIPT_SHA256 = (
    "b4b2623eabde7dd6b9acc250357a9f1ad61fa342c1dd496dcb634d4bfaca4d15"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):
    print("\n" + "=" * 112)
    print(text)
    print("=" * 112)


def run(cmd, *, env=None, check=True):

    p = subprocess.run(
        cmd,
        cwd=REPO,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            + " ".join(cmd)
            + "\n\n"
            + p.stdout
        )

    return p


def git(*args, env=None, check=True):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        check=check,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def git_blob_bytes(ref, relpath):

    p = subprocess.run(
        [
            "git",
            "show",
            f"{ref}:{relpath}",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stderr.decode(
                "utf-8",
                errors="replace",
            )
        )

    return p.stdout


# =============================================================================
# 2. LOCAL HASH GATE
# =============================================================================

banner(
    "STAGE26-3A-GIT :: LOCAL HASH GATE"
)


local_checks = [
    (
        "memory plan",
        PLAN_REL,
        EXPECTED_PLAN_SHA256,
    ),
    (
        "deployment package map",
        PACKAGE_MAP_REL,
        EXPECTED_PACKAGE_MAP_SHA256,
    ),
    (
        "memory worker",
        WORKER_REL,
        EXPECTED_WORKER_SHA256,
    ),
    (
        "implementation",
        IMPLEMENTATION_REL,
        EXPECTED_IMPLEMENTATION_SHA256,
    ),
    (
        "freeze receipt",
        FREEZE_RECEIPT_REL,
        EXPECTED_FREEZE_RECEIPT_SHA256,
    ),
    (
        "package manifest",
        MANIFEST_REL,
        EXPECTED_MANIFEST_SHA256,
    ),
]


for name, relpath, expected in local_checks:

    path = (
        REPO
        / relpath
    )

    if not path.exists():

        raise FileNotFoundError(
            path
        )

    actual = sha256_file(
        path
    )

    ok = (
        actual
        ==
        expected
    )

    print(
        f"{name:25s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual}"
    )

    if not ok:

        raise RuntimeError(
            f"Local Stage26-3A SHA mismatch: {name}"
        )


# =============================================================================
# 3. MANIFEST-COMPLETE LOCAL VERIFICATION
# =============================================================================

banner(
    "STAGE26-3A-GIT :: MANIFEST-COMPLETE LOCAL GATE"
)


manifest = json.loads(
    (
        REPO
        / MANIFEST_REL
    ).read_text(
        encoding="utf-8"
    )
)


if manifest.get(
    "status"
) != "READY_FOR_GIT_ANCHOR":

    raise RuntimeError(
        "Memory lock package is not READY_FOR_GIT_ANCHOR."
    )


if manifest.get(
    "parent_commit"
) != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-3A manifest parent mismatch."
    )


manifest_files = manifest[
    "files"
]


print(
    "Manifest-listed files:",
    len(
        manifest_files
    ),
)


if len(
    manifest_files
) != 5:

    raise RuntimeError(
        "Expected exactly five manifest-listed lock artifacts."
    )


for record in manifest_files:

    relpath = record[
        "path"
    ]

    path = (
        REPO
        / relpath
    )

    actual_sha = sha256_file(
        path
    )

    actual_size = int(
        path.stat().st_size
    )


    sha_ok = (
        actual_sha
        ==
        record[
            "sha256"
        ]
    )

    size_ok = (
        actual_size
        ==
        int(
            record[
                "size_bytes"
            ]
        )
    )


    print(
        f"{relpath:110s} "
        f"sha={'PASS' if sha_ok else 'FAIL'} "
        f"size={'PASS' if size_ok else 'FAIL'}"
    )


    if not sha_ok or not size_ok:

        raise RuntimeError(
            f"Manifest mismatch: {relpath}"
        )


# =============================================================================
# 4. SCIENTIFIC FREEZE RECEIPT GATE
# =============================================================================

banner(
    "STAGE26-3A-GIT :: SCIENTIFIC FREEZE GATE"
)


freeze_receipt = json.loads(
    (
        REPO
        / FREEZE_RECEIPT_REL
    ).read_text(
        encoding="utf-8"
    )
)


expected_fields = {
    "status":
        "PASS",

    "source_cpu_condition_count":
        80,

    "memory_repetitions_per_condition":
        5,

    "planned_memory_observation_count":
        400,

    "memory_observations_performed":
        0,

    "model_deserialization_performed":
        False,

    "inference_performed":
        False,

    "timing_performed":
        False,

    "gpu_used":
        False,

    "holdout_reopened":
        False,
}


for key, expected in expected_fields.items():

    actual = freeze_receipt.get(
        key
    )

    print(
        f"{key:40s}: {actual!r}"
    )


    if actual != expected:

        raise RuntimeError(
            f"Freeze receipt mismatch: "
            f"{key}={actual!r}, expected={expected!r}"
        )


if freeze_receipt.get(
    "parent_commit"
) != EXPECTED_PARENT:

    raise RuntimeError(
        "Freeze receipt parent mismatch."
    )


print(
    "\n[PASS] Stage26-3 memory profiling remains pre-observation."
)


# =============================================================================
# 5. MEMORY PLAN STRUCTURAL GATE
# =============================================================================

banner(
    "STAGE26-3A-GIT :: 400-REPETITION MEMORY PLAN GATE"
)


memory_plan = json.loads(
    (
        REPO
        / PLAN_REL
    ).read_text(
        encoding="utf-8"
    )
)


if memory_plan.get(
    "frozen_before_first_memory_observation"
) is not True:

    raise RuntimeError(
        "Memory plan not marked pre-observation."
    )


if int(
    memory_plan.get(
        "condition_count",
        -1,
    )
) != 80:

    raise RuntimeError(
        "Memory plan condition count mismatch."
    )


if int(
    memory_plan.get(
        "repetitions_per_condition",
        -1,
    )
) != 5:

    raise RuntimeError(
        "Memory repetition count mismatch."
    )


if int(
    memory_plan.get(
        "total_memory_repetitions",
        -1,
    )
) != 400:

    raise RuntimeError(
        "Expected 400 memory repetitions."
    )


rows = memory_plan[
    "memory_repetitions"
]


if len(
    rows
) != 400:

    raise RuntimeError(
        "Memory repetition table is not length 400."
    )


execution_orders = sorted(
    int(
        row[
            "memory_execution_order"
        ]
    )
    for row in rows
)


if execution_orders != list(
    range(
        1,
        401,
    )
):

    raise RuntimeError(
        "Memory execution orders are not exactly 1..400."
    )


# Verify every frozen CPU condition appears exactly five times.
counts = {}


for row in rows:

    key = int(
        row[
            "source_cpu_execution_order"
        ]
    )

    counts[
        key
    ] = (
        counts.get(
            key,
            0,
        )
        +
        1
    )


if set(
    counts
) != set(
    range(
        1,
        81,
    )
):

    raise RuntimeError(
        "Memory plan does not cover all 80 CPU conditions."
    )


if any(
    count != 5
    for count in counts.values()
):

    raise RuntimeError(
        "Not every CPU condition appears exactly five times."
    )


print(
    "[PASS] 80 frozen CPU conditions × 5 fresh repetitions = 400."
)


# =============================================================================
# 6. UPSTREAM LOCAL IMMUTABILITY GATE
# =============================================================================

banner(
    "STAGE26-3A-GIT :: UPSTREAM LOCAL IMMUTABILITY"
)


upstream_checks = [
    (
        "measurement protocol",
        PROTOCOL_REL,
        EXPECTED_PROTOCOL_SHA256,
    ),
    (
        "CPU execution plan",
        CPU_PLAN_REL,
        EXPECTED_CPU_PLAN_SHA256,
    ),
    (
        "Stage26-2 warm receipt",
        STAGE26_2_RECEIPT_REL,
        EXPECTED_STAGE26_2_RECEIPT_SHA256,
    ),
]


for name, relpath, expected in upstream_checks:

    actual = sha256_file(
        REPO
        / relpath
    )

    ok = (
        actual
        ==
        expected
    )

    print(
        f"{name:28s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual}"
    )

    if not ok:

        raise RuntimeError(
            f"Upstream artifact changed: {name}"
        )


# =============================================================================
# 7. CURRENT GIT STATE / IDEMPOTENCY
# =============================================================================

banner(
    "STAGE26-3A-GIT :: CURRENT GIT STATE"
)


HEAD_BEFORE = git(
    "rev-parse",
    "HEAD",
)

STATUS_BEFORE = git(
    "status",
    "--porcelain",
)


print(
    "Current HEAD:",
    HEAD_BEFORE,
)

print(
    "Repository clean:",
    STATUS_BEFORE == "",
)


if STATUS_BEFORE:

    print(
        "\nRepository status:"
    )

    print(
        STATUS_BEFORE
    )


already_committed = False


if HEAD_BEFORE == EXPECTED_PARENT:

    already_committed = False


else:

    parent = git(
        "rev-parse",
        "HEAD^",
        check=False,
    )

    subject = git(
        "log",
        "-1",
        "--pretty=%s",
    )


    if (
        parent == EXPECTED_PARENT
        and
        subject == COMMIT_MESSAGE
    ):

        already_committed = True

        print(
            "\n[INFO] Expected Stage26-3A lock commit "
            "already exists locally."
        )


    else:

        raise RuntimeError(
            "HEAD is neither the Stage26-2 parent nor the "
            "expected Stage26-3A child commit."
        )


# =============================================================================
# 8. STAGE + COMMIT
# =============================================================================

if not already_committed:

    banner(
        "STAGE26-3A-GIT :: STAGE + COMMIT"
    )


    if not STATUS_BEFORE:

        raise RuntimeError(
            "Expected uncommitted Stage26-3A lock package."
        )


    unexpected = []


    for line in STATUS_BEFORE.splitlines():

        relpath = line[
            3:
        ]


        if not relpath.startswith(
            LOCK_REL
            + "/"
        ):

            unexpected.append(
                line
            )


    if unexpected:

        raise RuntimeError(
            "Unexpected modifications outside Stage26-3A:\n"
            +
            "\n".join(
                unexpected
            )
        )


    git(
        "add",
        "--",
        LOCK_REL,
    )


    staged = git(
        "diff",
        "--cached",
        "--name-status",
    )


    print(
        staged
    )


    if not staged:

        raise RuntimeError(
            "Nothing staged."
        )


    for line in staged.splitlines():

        relpath = line.split(
            "\t"
        )[
            -1
        ]


        if not relpath.startswith(
            LOCK_REL
            + "/"
        ):

            raise RuntimeError(
                "Unexpected staged file:\n"
                + line
            )


    if not git(
        "config",
        "--get",
        "user.name",
        check=False,
    ):

        git(
            "config",
            "user.name",
            "themubasshir",
        )


    if not git(
        "config",
        "--get",
        "user.email",
        check=False,
    ):

        git(
            "config",
            "user.email",
            "themubasshir@users.noreply.github.com",
        )


    print(
        git(
            "commit",
            "-m",
            COMMIT_MESSAGE,
        )
    )


    STAGE26_3A_HEAD = git(
        "rev-parse",
        "HEAD",
    )


    if git(
        "rev-parse",
        "HEAD^",
    ) != EXPECTED_PARENT:

        raise RuntimeError(
            "Stage26-3A commit parent mismatch."
        )


else:

    STAGE26_3A_HEAD = HEAD_BEFORE


print(
    "\nStage26-3A commit:",
    STAGE26_3A_HEAD,
)


# =============================================================================
# 9. COMMITTED BLOB VERIFICATION
# =============================================================================

banner(
    "STAGE26-3A-GIT :: COMMITTED BLOB GATE"
)


manifest_blob_sha = sha256_bytes(
    git_blob_bytes(
        "HEAD",
        MANIFEST_REL,
    )
)


if manifest_blob_sha != EXPECTED_MANIFEST_SHA256:

    raise RuntimeError(
        "Committed package manifest mismatch."
    )


print(
    "package manifest PASS",
    manifest_blob_sha,
)


for record in manifest_files:

    relpath = record[
        "path"
    ]

    expected = record[
        "sha256"
    ]


    actual = sha256_bytes(
        git_blob_bytes(
            "HEAD",
            relpath,
        )
    )


    ok = (
        actual
        ==
        expected
    )


    print(
        f"{relpath:110s} "
        f"{'PASS' if ok else 'FAIL'}"
    )


    if not ok:

        raise RuntimeError(
            f"Committed lock blob mismatch: {relpath}"
        )


# =============================================================================
# 10. FETCH REMOTE RELATIONSHIP
# =============================================================================

banner(
    "STAGE26-3A-GIT :: REMOTE RELATIONSHIP"
)


git(
    "fetch",
    "origin",
    "main",
)


REMOTE_BEFORE = git(
    "rev-parse",
    "origin/main",
)


print(
    "Local Stage26-3A HEAD:",
    STAGE26_3A_HEAD,
)

print(
    "origin/main currently:",
    REMOTE_BEFORE,
)


if REMOTE_BEFORE == STAGE26_3A_HEAD:

    relationship = (
        "REMOTE_ALREADY_HAS_STAGE26_3A"
    )


elif REMOTE_BEFORE == EXPECTED_PARENT:

    relationship = (
        "LOCAL_STAGE26_3A_AHEAD_BY_ONE"
    )


else:

    relationship = (
        "UNEXPECTED_REMOTE_DIVERGENCE"
    )


print(
    "Relationship:",
    relationship,
)


if relationship == "UNEXPECTED_REMOTE_DIVERGENCE":

    raise RuntimeError(
        "origin/main diverged unexpectedly. "
        "Do not force push."
    )


# =============================================================================
# 11. PUSH IF NEEDED
# =============================================================================

if relationship == "LOCAL_STAGE26_3A_AHEAD_BY_ONE":

    banner(
        "STAGE26-3A-GIT :: PUSH"
    )


    secrets = UserSecretsClient()


    aliases = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "gh_token",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
        "gh_pat",
    ]


    token = None
    token_label = None


    for label in aliases:

        try:

            value = secrets.get_secret(
                label
            )

        except Exception:

            value = None


        if value:

            token = value.strip()
            token_label = label

            break


    if not token:

        raise RuntimeError(
            "No usable GitHub secret found."
        )


    print(
        f"[FOUND] {token_label} "
        f"({len(token)} characters)"
    )

    print(
        "Token value will not be printed."
    )


    fd, askpass_path = tempfile.mkstemp(
        prefix="stage26_3a_askpass_",
        suffix=".sh",
    )

    os.close(
        fd
    )


    askpass = Path(
        askpass_path
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' "x-access-token" ;;
  *Password*) printf '%s\\n' "$GITHUB_TOKEN" ;;
  *)          printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        stat.S_IRUSR
        |
        stat.S_IWUSR
        |
        stat.S_IXUSR
    )


    env = os.environ.copy()

    env[
        "GITHUB_TOKEN"
    ] = token

    env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"


    try:

        result = run(
            [
                "git",
                "push",
                "origin",
                "HEAD:main",
            ],
            env=env,
        )

        print(
            result.stdout
        )


    finally:

        try:
            askpass.unlink()

        except FileNotFoundError:
            pass


        token = None


# =============================================================================
# 12. FETCH-BACK + INDEPENDENT REF VERIFICATION
# =============================================================================

banner(
    "STAGE26-3A-GIT :: REMOTE COMMIT GATE"
)


git(
    "fetch",
    "origin",
    "main",
)


LOCAL_HEAD = git(
    "rev-parse",
    "HEAD",
)

REMOTE_HEAD = git(
    "rev-parse",
    "origin/main",
)


ls_remote = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


ls_remote_sha = (
    ls_remote.split()[0]
    if ls_remote
    else None
)


print(
    "Local HEAD :",
    LOCAL_HEAD,
)

print(
    "Remote HEAD:",
    REMOTE_HEAD,
)

print(
    "ls-remote :",
    ls_remote_sha,
)


if not (
    LOCAL_HEAD
    ==
    REMOTE_HEAD
    ==
    ls_remote_sha
    ==
    STAGE26_3A_HEAD
):

    raise RuntimeError(
        "Local/remote Stage26-3A commit mismatch."
    )


# =============================================================================
# 13. REMOTE BYTE-FOR-BYTE PACKAGE VERIFICATION
# =============================================================================

banner(
    "STAGE26-3A-GIT :: REMOTE PACKAGE HASH GATE"
)


remote_manifest_sha = sha256_bytes(
    git_blob_bytes(
        "origin/main",
        MANIFEST_REL,
    )
)


if remote_manifest_sha != EXPECTED_MANIFEST_SHA256:

    raise RuntimeError(
        "Remote Stage26-3A manifest mismatch."
    )


print(
    "package manifest PASS",
    remote_manifest_sha,
)


for index, record in enumerate(
    manifest_files,
    start=1,
):

    relpath = record[
        "path"
    ]

    expected = record[
        "sha256"
    ]


    actual = sha256_bytes(
        git_blob_bytes(
            "origin/main",
            relpath,
        )
    )


    if actual != expected:

        raise RuntimeError(
            f"Remote lock artifact mismatch: {relpath}"
        )


    print(
        f"[{index:02d}/{len(manifest_files):02d}] "
        f"PASS {relpath}"
    )


# =============================================================================
# 14. EXPLICIT REMOTE LOCK HASHES
# =============================================================================

banner(
    "STAGE26-3A-GIT :: EXPLICIT REMOTE HASHES"
)


explicit_checks = [
    (
        "memory plan",
        PLAN_REL,
        EXPECTED_PLAN_SHA256,
    ),
    (
        "package map",
        PACKAGE_MAP_REL,
        EXPECTED_PACKAGE_MAP_SHA256,
    ),
    (
        "worker",
        WORKER_REL,
        EXPECTED_WORKER_SHA256,
    ),
    (
        "implementation",
        IMPLEMENTATION_REL,
        EXPECTED_IMPLEMENTATION_SHA256,
    ),
    (
        "freeze receipt",
        FREEZE_RECEIPT_REL,
        EXPECTED_FREEZE_RECEIPT_SHA256,
    ),
]


for name, relpath, expected in explicit_checks:

    actual = sha256_bytes(
        git_blob_bytes(
            "origin/main",
            relpath,
        )
    )


    ok = (
        actual
        ==
        expected
    )


    print(
        f"{name:20s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual}"
    )


    if not ok:

        raise RuntimeError(
            f"Remote explicit hash failed: {name}"
        )


# =============================================================================
# 15. REMOTE UPSTREAM IMMUTABILITY
# =============================================================================

banner(
    "STAGE26-3A-GIT :: UPSTREAM REMOTE IMMUTABILITY"
)


for name, relpath, expected in upstream_checks:

    actual = sha256_bytes(
        git_blob_bytes(
            "origin/main",
            relpath,
        )
    )


    ok = (
        actual
        ==
        expected
    )


    print(
        f"{name:28s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual}"
    )


    if not ok:

        raise RuntimeError(
            f"Remote upstream lock changed: {name}"
        )


# =============================================================================
# 16. FINAL AUDIT
# =============================================================================

banner(
    "STAGE26-3A-GIT :: FINAL AUDIT"
)


FINAL_STATUS = git(
    "status",
    "--porcelain",
)


print(
    "Repository clean:",
    FINAL_STATUS == "",
)


if FINAL_STATUS:

    print(
        FINAL_STATUS
    )

    raise RuntimeError(
        "Repository dirty after Stage26-3A anchor."
    )


if git(
    "rev-parse",
    "HEAD^",
) != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-3A final parent mismatch."
    )


if git(
    "log",
    "-1",
    "--pretty=%s",
) != COMMIT_MESSAGE:

    raise RuntimeError(
        "Stage26-3A commit subject mismatch."
    )


# =============================================================================
# 17. CLOSURE
# =============================================================================

banner(
    "STAGE26-3A-GIT COMPLETE"
)


print(
    "STAGE26-2 PARENT:"
)

print(
    " ",
    EXPECTED_PARENT,
)


print(
    "\nSTAGE26-3A MEMORY LOCK COMMIT:"
)

print(
    " ",
    STAGE26_3A_HEAD,
)


print(
    "\nREMOTE MAIN:"
)

print(
    " ",
    REMOTE_HEAD,
)


print(
    "\nFROZEN MEMORY DESIGN:"
)

print(
    "  SOURCE CPU CONDITIONS       : 80"
)

print(
    "  FRESH REPS / CONDITION      : 5"
)

print(
    "  TOTAL MEMORY REPETITIONS    : 400"
)

print(
    "  RSS SAMPLE INTERVAL         : 5 ms"
)

print(
    "  MEMORY INFERENCE PASSES     : 1 / repetition"
)

print(
    "  MAX WORKER RAM FRACTION     : 0.85"
)


print(
    "\nMEMORY PLAN SHA256:"
)

print(
    " ",
    EXPECTED_PLAN_SHA256,
)


print(
    "\nDEPLOYMENT PACKAGE MAP SHA256:"
)

print(
    " ",
    EXPECTED_PACKAGE_MAP_SHA256,
)


print(
    "\nMEMORY WORKER SHA256:"
)

print(
    " ",
    EXPECTED_WORKER_SHA256,
)


print(
    "\nIMPLEMENTATION SHA256:"
)

print(
    " ",
    EXPECTED_IMPLEMENTATION_SHA256,
)


print(
    "\nFREEZE RECEIPT SHA256:"
)

print(
    " ",
    EXPECTED_FREEZE_RECEIPT_SHA256,
)


print(
    "\nPACKAGE MANIFEST SHA256:"
)

print(
    " ",
    EXPECTED_MANIFEST_SHA256,
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  COMMIT MATCH               : PASS"
)

print(
    "  LS-REMOTE                  : PASS"
)

print(
    "  ALL LOCK ARTIFACTS         : PASS"
)

print(
    "  ORIGINAL PROTOCOL          : PASS"
)

print(
    "  ORIGINAL CPU PLAN          : PASS"
)

print(
    "  STAGE26-2 RECEIPT          : PASS"
)

print(
    "  REPOSITORY CLEAN           : PASS"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  MEMORY IMPLEMENTATION DURABLY FROZEN BEFORE OBSERVATION"
)

print(
    "  MEMORY OBSERVATIONS = 0"
)

print(
    "  MODEL LOADS = 0"
)

print(
    "  INFERENCE = 0"
)

print(
    "  PERFORMANCE TIMING = 0"
)

print(
    "  GPU NOT USED"
)

print(
    "  HOLDOUT NOT REOPENED"
)


print(
    "\nNEXT:"
)

print(
    "  STAGE26-3B — EXECUTE THE FROZEN 400-REPETITION"
)

print(
    "  CPU MEMORY PROFILE."
)

print(
    "  GPU REMAINS OFF."
)


STAGE26-3A-GIT :: LOCAL HASH GATE
memory plan               PASS 018584b8f8c94d6afa5fcf3576c261bae2b0f694c8b748bf2e0421234e17ea13
deployment package map    PASS 09e0d093a8ca83f4d68e9affd675af4336b982a6af14e5034fc2e5ffd4bc547c
memory worker             PASS 2ba87c88244b8b3d8b0265d44b29204dc87e321e5aedc35dc5ac536de5fa8fd7
implementation            PASS 9e8439d41924f5bdd13dc4db0a8e997fb9a6be1883a38a9ff4028c9f785452dc
freeze receipt            PASS 465cc0c89f09c42f0d33b4dc1f8e759983fccd2897cef0207c85e777cdee3a19
package manifest          PASS de1b2731a276d4628360478f704ac16feabcad97925f3f1079657f4aced24eb1

STAGE26-3A-GIT :: MANIFEST-COMPLETE LOCAL GATE
Manifest-listed files: 5
results/stage26_deployment_profiling/stage26_3a_memory_implementation_lock/stage26_3_deployment_package_map.json sha=PASS size=PASS
results/stage26_deployment_profiling/stage26_3a_memory_implementation_lock/stage26_3_memory_execution_plan.json sha=PASS size=PASS
results/stage26_deployment_profiling/stage26_3a_memor

In [23]:
# =============================================================================
# STAGE26-3B — EXECUTE FROZEN CPU MEMORY + PACKAGE FOOTPRINT PROFILE
#
# PARENT:
#   2e1ece6cc198b054b01e467275951caafb6af794
#
# PRE-FROZEN DESIGN:
#   80 CPU conditions
#   × 5 fresh-process repetitions
#   = 400 memory repetitions
#
# MEMORY WORKER:
#   already frozen + git-anchored
#
# MEMORY METRICS:
#   baseline RSS
#   loaded RSS
#   peak RSS
#   delta model RSS
#   delta peak RSS
#   ru_maxrss descriptive
#
# EACH REPETITION:
#   fresh Python subprocess
#   exactly ONE untimed inference pass
#   internal RSS sampling every 5 ms
#
# PARENT RESOURCE ENFORCEMENT:
#   worker RSS > 85% physical RAM -> RESOURCE_LIMIT_OOM
#   600 s global frozen resource timeout -> TIMEOUT_RESOURCE_LIMIT
#
# IMPORTANT:
#   - NO latency/performance timing is collected.
#   - Parent wall-clock is used ONLY for timeout/heartbeat control.
#   - NO batch substitution.
#   - NO condition omission based on Stage26-2.
#   - GPU OFF.
#   - HOLDOUT CLOSED.
#
# RESUME:
#   Existing valid final memory observation receipts are NEVER re-run.
# =============================================================================

from __future__ import annotations

import os
import sys
import csv
import json
import time
import signal
import shutil
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import psutil


# =============================================================================
# 0. PATHS / FROZEN IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

MEMORY_ROOT = (
    STAGE26_ROOT
    / "memory"
)

OBS_DIR = (
    MEMORY_ROOT
    / "observations"
)

CONFIG_DIR = (
    MEMORY_ROOT
    / "configs"
)

LOCK_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_3a_memory_implementation_lock"
)

RESULT_PACKAGE_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_3b_cpu_memory_profile"
)


OBS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


EXPECTED_HEAD = (
    "2e1ece6cc198b054b01e467275951caafb6af794"
)

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

EXPECTED_CPU_PLAN_SHA256 = (
    "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363"
)

EXPECTED_STAGE26_2_RECEIPT_SHA256 = (
    "b4b2623eabde7dd6b9acc250357a9f1ad61fa342c1dd496dcb634d4bfaca4d15"
)

EXPECTED_MEMORY_PLAN_SHA256 = (
    "018584b8f8c94d6afa5fcf3576c261bae2b0f694c8b748bf2e0421234e17ea13"
)

EXPECTED_PACKAGE_MAP_SHA256 = (
    "09e0d093a8ca83f4d68e9affd675af4336b982a6af14e5034fc2e5ffd4bc547c"
)

EXPECTED_WORKER_SHA256 = (
    "2ba87c88244b8b3d8b0265d44b29204dc87e321e5aedc35dc5ac536de5fa8fd7"
)

EXPECTED_IMPLEMENTATION_SHA256 = (
    "9e8439d41924f5bdd13dc4db0a8e997fb9a6be1883a38a9ff4028c9f785452dc"
)

EXPECTED_FREEZE_RECEIPT_SHA256 = (
    "465cc0c89f09c42f0d33b4dc1f8e759983fccd2897cef0207c85e777cdee3a19"
)

EXPECTED_LOCK_MANIFEST_SHA256 = (
    "de1b2731a276d4628360478f704ac16feabcad97925f3f1079657f4aced24eb1"
)


PROTOCOL = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_0_protocol_lock/"
      "measurement_protocol.json"
)

CPU_PLAN = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_0_protocol_lock/"
      "cpu_execution_plan.json"
)

STAGE26_2_RECEIPT = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_2_cpu_warm_inference/"
      "stage26_2_warm_cpu_receipt.json"
)

MEMORY_PLAN = (
    LOCK_DIR
    / "stage26_3_memory_execution_plan.json"
)

PACKAGE_MAP = (
    LOCK_DIR
    / "stage26_3_deployment_package_map.json"
)

WORKER = (
    LOCK_DIR
    / "stage26_memory_worker.py"
)

IMPLEMENTATION = (
    LOCK_DIR
    / "stage26_3_memory_implementation.json"
)

FREEZE_RECEIPT = (
    LOCK_DIR
    / "stage26_3a_memory_freeze_receipt.json"
)

LOCK_MANIFEST = (
    LOCK_DIR
    / "stage26_3a_memory_lock_package_manifest.json"
)


# =============================================================================
# 1. EXECUTION CONSTANTS
# =============================================================================

MEASUREMENT_SEED = 26042

RSS_PARENT_SAMPLE_SECONDS = 0.005

MAX_WORKER_RAM_FRACTION = 0.85

CONDITION_TIMEOUT_SECONDS = 600

HEARTBEAT_SECONDS = 30

CPU_UTILIZATION_GATE_PERCENT = 20.0

MIN_AVAILABLE_RAM_GIB = 8.0

ENVIRONMENT_RETRY_COUNT = 3

ENVIRONMENT_RETRY_COOLDOWN_SECONDS = 5

CPU_UTILIZATION_SAMPLE_SECONDS = 1.0


FINAL_STATUSES = {
    "PASS",
    "RESOURCE_LIMIT_OOM",
    "TIMEOUT_RESOURCE_LIMIT",
}


# =============================================================================
# 2. AGGREGATE OUTPUTS
# =============================================================================

RAW_JSONL = (
    MEMORY_ROOT
    / "stage26_3_memory_raw.jsonl"
)

RAW_CSV = (
    MEMORY_ROOT
    / "stage26_3_memory_raw.csv"
)

SUMMARY_CSV = (
    MEMORY_ROOT
    / "stage26_3_memory_summary.csv"
)

SUMMARY_JSON = (
    MEMORY_ROOT
    / "stage26_3_memory_summary.json"
)

PACKAGE_SIZE_CSV = (
    MEMORY_ROOT
    / "stage26_3_package_sizes.csv"
)

RECEIPT_PATH = (
    MEMORY_ROOT
    / "stage26_3b_memory_receipt.json"
)

EXECUTION_RECORD = (
    MEMORY_ROOT
    / "stage26_3b_execution_record.json"
)

FAILURE_MARKER = (
    MEMORY_ROOT
    / "stage26_3b_worker_failure.json"
)

ENV_FAILURE_MARKER = (
    MEMORY_ROOT
    / "stage26_3b_invalid_environment.json"
)


# =============================================================================
# 3. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 116
    )

    print(text)

    print(
        "=" * 116
    )


def git(*args):

    p = subprocess.run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def mib(value):

    if value is None:
        return None

    return float(
        value
    ) / 1024**2


def environment_gate():

    attempts = []

    for attempt in range(
        1,
        ENVIRONMENT_RETRY_COUNT + 1,
    ):

        cpu_percent = float(
            psutil.cpu_percent(
                interval=CPU_UTILIZATION_SAMPLE_SECONDS
            )
        )

        available_ram_gib = float(
            psutil.virtual_memory().available
            /
            1024**3
        )

        passed = (
            cpu_percent
            <=
            CPU_UTILIZATION_GATE_PERCENT
            and
            available_ram_gib
            >=
            MIN_AVAILABLE_RAM_GIB
        )

        record = {
            "attempt":
                attempt,

            "cpu_percent":
                cpu_percent,

            "available_ram_gib":
                available_ram_gib,

            "passed":
                passed,
        }

        attempts.append(
            record
        )

        if passed:

            return (
                True,
                attempts,
                record,
            )

        if attempt < ENVIRONMENT_RETRY_COUNT:

            time.sleep(
                ENVIRONMENT_RETRY_COOLDOWN_SECONDS
            )

    return (
        False,
        attempts,
        attempts[
            -1
        ],
    )


# =============================================================================
# 4. PARENT + IMMUTABILITY GATE
# =============================================================================

banner(
    "STAGE26-3B :: PARENT / IMMUTABILITY GATE"
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD :",
    EXPECTED_HEAD,
)

print(
    "Local HEAD    :",
    head,
)

print(
    "origin/main   :",
    remote,
)

print(
    "Repository clean:",
    status == "",
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected local HEAD."
    )


if remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed after Stage26-3A."
    )


if status:

    raise RuntimeError(
        "Repository must be clean before Stage26-3B:\n"
        +
        status
    )


immutable_checks = [
    (
        "measurement protocol",
        PROTOCOL,
        EXPECTED_PROTOCOL_SHA256,
    ),
    (
        "CPU plan",
        CPU_PLAN,
        EXPECTED_CPU_PLAN_SHA256,
    ),
    (
        "Stage26-2 receipt",
        STAGE26_2_RECEIPT,
        EXPECTED_STAGE26_2_RECEIPT_SHA256,
    ),
    (
        "memory plan",
        MEMORY_PLAN,
        EXPECTED_MEMORY_PLAN_SHA256,
    ),
    (
        "package map",
        PACKAGE_MAP,
        EXPECTED_PACKAGE_MAP_SHA256,
    ),
    (
        "memory worker",
        WORKER,
        EXPECTED_WORKER_SHA256,
    ),
    (
        "memory implementation",
        IMPLEMENTATION,
        EXPECTED_IMPLEMENTATION_SHA256,
    ),
    (
        "freeze receipt",
        FREEZE_RECEIPT,
        EXPECTED_FREEZE_RECEIPT_SHA256,
    ),
    (
        "lock manifest",
        LOCK_MANIFEST,
        EXPECTED_LOCK_MANIFEST_SHA256,
    ),
]


for name, path, expected_sha in immutable_checks:

    actual_sha = sha256_file(
        path
    )

    ok = (
        actual_sha
        ==
        expected_sha
    )

    print(
        f"{name:28s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual_sha}"
    )

    if not ok:

        raise RuntimeError(
            f"Frozen artifact changed: {name}"
        )


# =============================================================================
# 5. LOAD FROZEN 400-REPETITION PLAN
# =============================================================================

banner(
    "STAGE26-3B :: LOAD FROZEN MEMORY PLAN"
)


memory_plan = json.loads(
    MEMORY_PLAN.read_text(
        encoding="utf-8"
    )
)


rows = sorted(
    memory_plan[
        "memory_repetitions"
    ],
    key=lambda row: int(
        row[
            "memory_execution_order"
        ]
    ),
)


if len(
    rows
) != 400:

    raise RuntimeError(
        "Expected exactly 400 frozen memory repetitions."
    )


if [
    int(
        row[
            "memory_execution_order"
        ]
    )
    for row in rows
] != list(
    range(
        1,
        401,
    )
):

    raise RuntimeError(
        "Memory execution order is not exactly 1..400."
    )


print(
    "Frozen memory repetitions:",
    len(
        rows
    ),
)


# =============================================================================
# 6. DOUBLE-RUN / FAILURE-MARKER GATE
# =============================================================================

banner(
    "STAGE26-3B :: RESUME / DOUBLE-RUN GATE"
)


if RESULT_PACKAGE_DIR.exists():

    raise RuntimeError(
        "Stage26-3B repository result package already exists. "
        "Do not remeasure."
    )


for marker in [
    FAILURE_MARKER,
    ENV_FAILURE_MARKER,
]:

    if marker.exists():

        print(
            "Existing non-resource failure marker:"
        )

        print(
            " ",
            marker
        )

        raise RuntimeError(
            "Stage26-3B previously stopped at a methodological/"
            "implementation/environment failure. "
            "Do not silently resume."
        )


existing_paths = sorted(
    OBS_DIR.glob(
        "memory_*.json"
    )
)


print(
    "Existing durable memory observations:",
    len(
        existing_paths
    ),
)


# =============================================================================
# 7. VALIDATE ANY EXISTING RECEIPTS
# =============================================================================

row_by_order = {
    int(
        row[
            "memory_execution_order"
        ]
    ):
        row
    for row in rows
}


completed_orders = set()


for path in existing_paths:

    receipt = json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )

    order = int(
        receipt[
            "memory_execution_order"
        ]
    )


    if order not in row_by_order:

        raise RuntimeError(
            f"Unknown memory execution order {order}."
        )


    frozen = row_by_order[
        order
    ]


    for field in [
        "source_cpu_execution_order",
        "condition_id",
        "target_id",
        "hardware_mode",
        "thread_count",
        "batch_size",
        "memory_repetition",
    ]:

        if receipt.get(
            field
        ) != frozen.get(
            field
        ):

            raise RuntimeError(
                f"Existing memory receipt {order:03d} "
                f"mismatch at {field}."
            )


    if receipt[
        "status"
    ] not in FINAL_STATUSES:

        raise RuntimeError(
            f"Existing memory receipt {order:03d} "
            f"has invalid status {receipt['status']}."
        )


    if receipt[
        "status"
    ] == "PASS":

        required_metrics = [
            "baseline_rss_bytes",
            "loaded_rss_bytes",
            "peak_rss_bytes",
            "delta_model_rss_bytes",
            "delta_peak_rss_bytes",
            "ru_maxrss_bytes_descriptive",
        ]


        for metric in required_metrics:

            if metric not in receipt:

                raise RuntimeError(
                    f"Existing PASS receipt {order:03d} "
                    f"missing {metric}."
                )


    completed_orders.add(
        order
    )


print(
    "Validated durable observations:",
    len(
        completed_orders
    ),
)


# =============================================================================
# 8. SYSTEM RAM CEILING
# =============================================================================

physical_ram_bytes = int(
    psutil.virtual_memory().total
)

worker_ram_limit_bytes = int(
    physical_ram_bytes
    *
    MAX_WORKER_RAM_FRACTION
)


banner(
    "STAGE26-3B :: RESOURCE ENVELOPE"
)


print(
    "Physical RAM:",
    f"{physical_ram_bytes / 1024**3:.3f} GiB",
)

print(
    "85% worker ceiling:",
    f"{worker_ram_limit_bytes / 1024**3:.3f} GiB",
)

print(
    "Parent RSS sampling:",
    "5 ms",
)

print(
    "Resource timeout:",
    "600 s",
)

print(
    "GPU:",
    "OFF",
)


# =============================================================================
# 9. EXECUTE 400 FROZEN MEMORY REPETITIONS
# =============================================================================

banner(
    "STAGE26-3B :: EXECUTE FROZEN MEMORY PLAN"
)


for frozen in rows:

    order = int(
        frozen[
            "memory_execution_order"
        ]
    )


    if order in completed_orders:

        print(
            f"[SKIP] {order:03d}/400 already durable"
        )

        continue


    target = frozen[
        "target_id"
    ]

    mode = frozen[
        "hardware_mode"
    ]

    batch_size = int(
        frozen[
            "batch_size"
        ]
    )

    threads = int(
        frozen[
            "thread_count"
        ]
    )

    affinity = [
        int(x)
        for x in frozen[
            "affinity"
        ]
    ]

    repetition = int(
        frozen[
            "memory_repetition"
        ]
    )


    # -------------------------------------------------------------------------
    # Frozen environment precondition gate.
    # -------------------------------------------------------------------------

    env_ok, env_attempts, env_final = (
        environment_gate()
    )


    if not env_ok:

        atomic_json(
            ENV_FAILURE_MARKER,
            {
                "schema":
                    "stage26_3b_invalid_environment_v1",

                "status":
                    "INVALID_ENVIRONMENT",

                "memory_execution_order":
                    order,

                "target_id":
                    target,

                "hardware_mode":
                    mode,

                "batch_size":
                    batch_size,

                "environment_attempts":
                    env_attempts,

                "completed_memory_orders":
                    sorted(
                        completed_orders
                    ),
            },
        )


        raise RuntimeError(
            "INVALID_ENVIRONMENT under frozen Stage26 rules."
        )


    result_path = (
        OBS_DIR
        / f"memory_{order:03d}.json"
    )

    config_path = (
        CONFIG_DIR
        / f"memory_{order:03d}.json"
    )


    config = {
        "schema":
            "stage26_3_memory_worker_config_v1",

        "repo":
            str(
                REPO
            ),

        "result_path":
            str(
                result_path
            ),

        "measurement_seed":
            MEASUREMENT_SEED,

        "memory_execution_order":
            order,

        "source_cpu_execution_order":
            int(
                frozen[
                    "source_cpu_execution_order"
                ]
            ),

        "condition_id":
            frozen[
                "condition_id"
            ],

        "target_id":
            target,

        "target_role":
            frozen[
                "target_role"
            ],

        "comparison_group":
            frozen[
                "comparison_group"
            ],

        "hardware_mode":
            mode,

        "thread_count":
            threads,

        "affinity":
            affinity,

        "batch_size":
            batch_size,

        "memory_repetition":
            repetition,
    }


    atomic_json(
        config_path,
        config,
    )


    env = os.environ.copy()

    thread_value = str(
        threads
    )


    for key in [
        "OMP_NUM_THREADS",
        "MKL_NUM_THREADS",
        "OPENBLAS_NUM_THREADS",
        "NUMEXPR_NUM_THREADS",
        "BLIS_NUM_THREADS",
        "VECLIB_MAXIMUM_THREADS",
    ]:

        env[
            key
        ] = thread_value


    env[
        "CUDA_VISIBLE_DEVICES"
    ] = ""


    print(
        f"\n[START] {order:03d}/400 "
        f"{target:43s} "
        f"{mode:21s} "
        f"B={batch_size:5d} "
        f"rep={repetition}",
        flush=True,
    )


    process = subprocess.Popen(
        [
            sys.executable,
            str(
                WORKER
            ),
            str(
                config_path
            ),
        ],
        cwd=REPO,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )


    child = psutil.Process(
        process.pid
    )

    operational_start = (
        time.monotonic()
    )

    last_heartbeat = (
        operational_start
    )

    parent_observed_peak_rss = 0

    ram_limit_triggered = False

    timed_out = False


    while True:

        returncode = process.poll()


        if returncode is not None:

            break


        # -------------------------------------------------------------
        # Parent-side worker RSS guard.
        # This is resource protection only, not the reported peak metric.
        # -------------------------------------------------------------

        try:

            current_rss = int(
                child.memory_info().rss
            )

            parent_observed_peak_rss = max(
                parent_observed_peak_rss,
                current_rss,
            )


            if current_rss > worker_ram_limit_bytes:

                ram_limit_triggered = True

                print(
                    f"[RAM LIMIT] {order:03d} "
                    f"worker RSS={current_rss / 1024**3:.3f} GiB "
                    f"> frozen 85% ceiling",
                    flush=True,
                )

                process.kill()

                break


        except psutil.NoSuchProcess:

            pass


        now = time.monotonic()

        operational_elapsed = (
            now
            -
            operational_start
        )


        # Global frozen resource timeout.
        if operational_elapsed >= CONDITION_TIMEOUT_SECONDS:

            timed_out = True

            print(
                f"[TIMEOUT] {order:03d} reached "
                f"frozen 600-second resource limit",
                flush=True,
            )

            process.kill()

            break


        if (
            now
            -
            last_heartbeat
        ) >= HEARTBEAT_SECONDS:

            last_heartbeat = now

            print(
                f"[HEARTBEAT {order:03d}] "
                f"still running; "
                f"worker_peak_seen="
                f"{parent_observed_peak_rss / 1024**3:.3f} GiB",
                flush=True,
            )


        time.sleep(
            RSS_PARENT_SAMPLE_SECONDS
        )


    stdout, stderr = (
        process.communicate()
    )


    # -------------------------------------------------------------------------
    # Parent-forced frozen resource outcomes.
    # -------------------------------------------------------------------------

    if ram_limit_triggered:

        atomic_json(
            result_path,
            {
                "schema":
                    "stage26_3_memory_observation_v1",

                "status":
                    "RESOURCE_LIMIT_OOM",

                "resource_limit_reason":
                    "PARENT_85_PERCENT_PHYSICAL_RAM_CEILING",

                "memory_execution_order":
                    order,

                "source_cpu_execution_order":
                    int(
                        frozen[
                            "source_cpu_execution_order"
                        ]
                    ),

                "condition_id":
                    frozen[
                        "condition_id"
                    ],

                "target_id":
                    target,

                "target_role":
                    frozen[
                        "target_role"
                    ],

                "comparison_group":
                    frozen[
                        "comparison_group"
                    ],

                "hardware_mode":
                    mode,

                "thread_count":
                    threads,

                "affinity_requested":
                    affinity,

                "batch_size":
                    batch_size,

                "memory_repetition":
                    repetition,

                "worker_ram_limit_bytes":
                    worker_ram_limit_bytes,

                "parent_observed_peak_rss_bytes":
                    parent_observed_peak_rss,

                "environment_attempts":
                    env_attempts,

                "environment_final":
                    env_final,

                "timing_performed":
                    False,

                "holdout_accessed":
                    False,

                "gpu_used":
                    False,
            },
        )


    elif timed_out:

        atomic_json(
            result_path,
            {
                "schema":
                    "stage26_3_memory_observation_v1",

                "status":
                    "TIMEOUT_RESOURCE_LIMIT",

                "resource_limit_reason":
                    "FROZEN_600_SECOND_TIMEOUT",

                "memory_execution_order":
                    order,

                "source_cpu_execution_order":
                    int(
                        frozen[
                            "source_cpu_execution_order"
                        ]
                    ),

                "condition_id":
                    frozen[
                        "condition_id"
                    ],

                "target_id":
                    target,

                "target_role":
                    frozen[
                        "target_role"
                    ],

                "comparison_group":
                    frozen[
                        "comparison_group"
                    ],

                "hardware_mode":
                    mode,

                "thread_count":
                    threads,

                "affinity_requested":
                    affinity,

                "batch_size":
                    batch_size,

                "memory_repetition":
                    repetition,

                "timeout_seconds":
                    CONDITION_TIMEOUT_SECONDS,

                "parent_observed_peak_rss_bytes":
                    parent_observed_peak_rss,

                "environment_attempts":
                    env_attempts,

                "environment_final":
                    env_final,

                "timing_performed":
                    False,

                "holdout_accessed":
                    False,

                "gpu_used":
                    False,
            },
        )


    # -------------------------------------------------------------------------
    # Worker ended without its own receipt.
    # -------------------------------------------------------------------------

    elif not result_path.exists():

        stderr_lower = (
            stderr
            or
            ""
        ).lower()


        oom_like = (
            process.returncode
            in {
                -signal.SIGKILL,
                137,
            }
            or
            "out of memory"
            in stderr_lower
            or
            "cannot allocate memory"
            in stderr_lower
            or
            "memoryerror"
            in stderr_lower
            or
            "bad alloc"
            in stderr_lower
        )


        if oom_like:

            atomic_json(
                result_path,
                {
                    "schema":
                        "stage26_3_memory_observation_v1",

                    "status":
                        "RESOURCE_LIMIT_OOM",

                    "resource_limit_reason":
                        "WORKER_OOM_OR_SIGKILL",

                    "memory_execution_order":
                        order,

                    "source_cpu_execution_order":
                        int(
                            frozen[
                                "source_cpu_execution_order"
                            ]
                        ),

                    "condition_id":
                        frozen[
                            "condition_id"
                        ],

                    "target_id":
                        target,

                    "target_role":
                        frozen[
                            "target_role"
                        ],

                    "comparison_group":
                        frozen[
                            "comparison_group"
                        ],

                    "hardware_mode":
                        mode,

                    "thread_count":
                        threads,

                    "affinity_requested":
                        affinity,

                    "batch_size":
                        batch_size,

                    "memory_repetition":
                        repetition,

                    "returncode":
                        process.returncode,

                    "parent_observed_peak_rss_bytes":
                        parent_observed_peak_rss,

                    "environment_attempts":
                        env_attempts,

                    "environment_final":
                        env_final,

                    "timing_performed":
                        False,

                    "holdout_accessed":
                        False,

                    "gpu_used":
                        False,
                },
            )


        else:

            atomic_json(
                FAILURE_MARKER,
                {
                    "schema":
                        "stage26_3b_worker_failure_v1",

                    "status":
                        "WORKER_FAILURE",

                    "memory_execution_order":
                        order,

                    "target_id":
                        target,

                    "hardware_mode":
                        mode,

                    "batch_size":
                        batch_size,

                    "returncode":
                        process.returncode,

                    "stdout":
                        stdout[
                            -8000:
                        ],

                    "stderr":
                        stderr[
                            -16000:
                        ],

                    "parent_observed_peak_rss_bytes":
                        parent_observed_peak_rss,

                    "environment_attempts":
                        env_attempts,
                },
            )


            print(
                "\nWorker stdout:"
            )

            print(
                stdout
            )

            print(
                "\nWorker stderr:"
            )

            print(
                stderr
            )


            raise RuntimeError(
                f"Stage26-3B worker failure at memory order {order:03d}."
            )


    # -------------------------------------------------------------------------
    # Normalize receipt with parent-side metadata.
    # -------------------------------------------------------------------------

    receipt = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


    receipt[
        "target_role"
    ] = frozen[
        "target_role"
    ]

    receipt[
        "comparison_group"
    ] = frozen[
        "comparison_group"
    ]

    receipt[
        "environment_attempts"
    ] = env_attempts

    receipt[
        "environment_final"
    ] = env_final

    receipt[
        "parent_observed_peak_rss_bytes"
    ] = parent_observed_peak_rss

    receipt[
        "worker_ram_limit_bytes"
    ] = worker_ram_limit_bytes


    atomic_json(
        result_path,
        receipt,
    )


    # -------------------------------------------------------------------------
    # Scientific identity gate.
    # -------------------------------------------------------------------------

    for field in [
        "source_cpu_execution_order",
        "condition_id",
        "target_id",
        "hardware_mode",
        "thread_count",
        "batch_size",
        "memory_repetition",
    ]:

        if receipt.get(
            field
        ) != frozen.get(
            field
        ):

            raise RuntimeError(
                f"Memory receipt {order:03d} mismatch at {field}."
            )


    if receipt[
        "status"
    ] not in FINAL_STATUSES:

        atomic_json(
            FAILURE_MARKER,
            {
                "schema":
                    "stage26_3b_worker_failure_v1",

                "status":
                    "WORKER_FAILURE",

                "memory_execution_order":
                    order,

                "reason":
                    (
                        "Unexpected worker receipt status: "
                        +
                        str(
                            receipt[
                                "status"
                            ]
                        )
                    ),

                "receipt":
                    receipt,
            },
        )

        raise RuntimeError(
            f"Unexpected memory status at {order:03d}."
        )


    if receipt[
        "status"
    ] == "PASS":

        if receipt.get(
            "timing_performed"
        ) is not False:

            raise RuntimeError(
                "Memory worker claims timing was performed."
            )


        if receipt.get(
            "gpu_used"
        ) is not False:

            raise RuntimeError(
                "GPU unexpectedly used in CPU memory profiling."
            )


        if receipt.get(
            "holdout_accessed"
        ) is not False:

            raise RuntimeError(
                "Holdout unexpectedly accessed."
            )


        print(
            f"[PASS] {order:03d}/400 "
            f"baseline={mib(receipt['baseline_rss_bytes']):9.2f} MiB "
            f"loaded={mib(receipt['loaded_rss_bytes']):9.2f} MiB "
            f"peak={mib(receipt['peak_rss_bytes']):10.2f} MiB "
            f"Δmodel={mib(receipt['delta_model_rss_bytes']):8.2f} MiB "
            f"Δpeak={mib(receipt['delta_peak_rss_bytes']):9.2f} MiB",
            flush=True,
        )


    elif receipt[
        "status"
    ] == "RESOURCE_LIMIT_OOM":

        print(
            f"[OOM] {order:03d}/400 "
            f"{target} "
            f"{mode} "
            f"B={batch_size}",
            flush=True,
        )


    elif receipt[
        "status"
    ] == "TIMEOUT_RESOURCE_LIMIT":

        print(
            f"[TIMEOUT] {order:03d}/400 "
            f"{target} "
            f"{mode} "
            f"B={batch_size}",
            flush=True,
        )


    completed_orders.add(
        order
    )


# =============================================================================
# 10. ALL-400 COMPLETENESS GATE
# =============================================================================

banner(
    "STAGE26-3B :: ALL-400 COMPLETENESS GATE"
)


observation_paths = [
    OBS_DIR
    / f"memory_{order:03d}.json"
    for order in range(
        1,
        401,
    )
]


for path in observation_paths:

    if not path.exists():

        raise RuntimeError(
            f"Missing memory observation: {path.name}"
        )


observations = [
    json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )
    for path in observation_paths
]


status_counts = {}


for observation in observations:

    status_counts[
        observation[
            "status"
        ]
    ] = (
        status_counts.get(
            observation[
                "status"
            ],
            0,
        )
        +
        1
    )


print(
    "Status counts:"
)


for key, value in sorted(
    status_counts.items()
):

    print(
        f"  {key:30s}: {value}"
    )


if sum(
    status_counts.values()
) != 400:

    raise RuntimeError(
        "Memory status counts do not sum to 400."
    )


# =============================================================================
# 11. WRITE RAW MEMORY TABLE
# =============================================================================

banner(
    "STAGE26-3B :: RAW MEMORY TABLE"
)


raw_rows = []


for obs in observations:

    raw_rows.append(
        {
            "memory_execution_order":
                int(
                    obs[
                        "memory_execution_order"
                    ]
                ),

            "source_cpu_execution_order":
                int(
                    obs[
                        "source_cpu_execution_order"
                    ]
                ),

            "condition_id":
                obs[
                    "condition_id"
                ],

            "target_id":
                obs[
                    "target_id"
                ],

            "target_role":
                obs.get(
                    "target_role"
                ),

            "comparison_group":
                obs.get(
                    "comparison_group"
                ),

            "hardware_mode":
                obs[
                    "hardware_mode"
                ],

            "thread_count":
                int(
                    obs[
                        "thread_count"
                    ]
                ),

            "batch_size":
                int(
                    obs[
                        "batch_size"
                    ]
                ),

            "memory_repetition":
                int(
                    obs[
                        "memory_repetition"
                    ]
                ),

            "status":
                obs[
                    "status"
                ],

            "baseline_rss_bytes":
                obs.get(
                    "baseline_rss_bytes"
                ),

            "loaded_rss_bytes":
                obs.get(
                    "loaded_rss_bytes"
                ),

            "peak_rss_bytes":
                obs.get(
                    "peak_rss_bytes"
                ),

            "delta_model_rss_bytes":
                obs.get(
                    "delta_model_rss_bytes"
                ),

            "delta_peak_rss_bytes":
                obs.get(
                    "delta_peak_rss_bytes"
                ),

            "ru_maxrss_bytes_descriptive":
                obs.get(
                    "ru_maxrss_bytes_descriptive"
                ),

            "rss_sample_count":
                obs.get(
                    "rss_sample_count"
                ),

            "parent_observed_peak_rss_bytes":
                obs.get(
                    "parent_observed_peak_rss_bytes"
                ),

            "worker_ram_limit_bytes":
                obs.get(
                    "worker_ram_limit_bytes"
                ),
        }
    )


with RAW_JSONL.open(
    "w",
    encoding="utf-8",
) as f:

    for row in raw_rows:

        f.write(
            json.dumps(
                row,
                sort_keys=True,
            )
            +
            "\n"
        )


raw_columns = list(
    raw_rows[
        0
    ].keys()
)


with RAW_CSV.open(
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=raw_columns,
    )

    writer.writeheader()

    writer.writerows(
        raw_rows
    )


print(
    "Raw observations written:",
    len(
        raw_rows
    ),
)


# =============================================================================
# 12. CONDITION-LEVEL SUMMARY — FIVE REPS PER CONDITION
# =============================================================================

banner(
    "STAGE26-3B :: CONDITION-LEVEL MEMORY SUMMARY"
)


cpu_plan = json.loads(
    CPU_PLAN.read_text(
        encoding="utf-8"
    )
)


source_conditions = sorted(
    cpu_plan[
        "conditions"
    ],
    key=lambda row: int(
        row[
            "execution_order"
        ]
    ),
)


summary_rows = []


for condition in source_conditions:

    source_order = int(
        condition[
            "execution_order"
        ]
    )


    matching = [
        obs
        for obs in observations
        if int(
            obs[
                "source_cpu_execution_order"
            ]
        )
        ==
        source_order
    ]


    if len(
        matching
    ) != 5:

        raise RuntimeError(
            f"CPU condition {source_order:03d} does not have exactly 5 memory reps."
        )


    pass_rows = [
        obs
        for obs in matching
        if obs[
            "status"
        ] == "PASS"
    ]


    condition_status_counts = {}


    for obs in matching:

        condition_status_counts[
            obs[
                "status"
            ]
        ] = (
            condition_status_counts.get(
                obs[
                    "status"
                ],
                0,
            )
            +
            1
        )


    summary = {
        "source_cpu_execution_order":
            source_order,

        "condition_id":
            condition[
                "condition_id"
            ],

        "target_id":
            condition[
                "target_id"
            ],

        "target_role":
            condition[
                "target_role"
            ],

        "comparison_group":
            condition[
                "comparison_group"
            ],

        "hardware_mode":
            condition[
                "hardware_mode"
            ],

        "thread_count":
            int(
                condition[
                    "thread_count"
                ]
            ),

        "batch_size":
            int(
                condition[
                    "batch_size"
                ]
            ),

        "planned_repetitions":
            5,

        "pass_repetitions":
            len(
                pass_rows
            ),

        "resource_limit_oom_repetitions":
            int(
                condition_status_counts.get(
                    "RESOURCE_LIMIT_OOM",
                    0,
                )
            ),

        "timeout_resource_limit_repetitions":
            int(
                condition_status_counts.get(
                    "TIMEOUT_RESOURCE_LIMIT",
                    0,
                )
            ),
    }


    for metric in [
        "baseline_rss_bytes",
        "loaded_rss_bytes",
        "peak_rss_bytes",
        "delta_model_rss_bytes",
        "delta_peak_rss_bytes",
        "ru_maxrss_bytes_descriptive",
    ]:

        values = np.asarray(
            [
                float(
                    obs[
                        metric
                    ]
                )
                for obs in pass_rows
            ],
            dtype=np.float64,
        )


        if values.size:

            summary[
                f"median_{metric}"
            ] = float(
                np.median(
                    values
                )
            )

            summary[
                f"min_{metric}"
            ] = float(
                np.min(
                    values
                )
            )

            summary[
                f"max_{metric}"
            ] = float(
                np.max(
                    values
                )
            )

        else:

            summary[
                f"median_{metric}"
            ] = None

            summary[
                f"min_{metric}"
            ] = None

            summary[
                f"max_{metric}"
            ] = None


    summary_rows.append(
        summary
    )


with SUMMARY_CSV.open(
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=list(
            summary_rows[
                0
            ].keys()
        ),
    )

    writer.writeheader()

    writer.writerows(
        summary_rows
    )


# =============================================================================
# 13. FROZEN PACKAGE FOOTPRINT TABLE
# =============================================================================

banner(
    "STAGE26-3B :: SERIALIZED / DEPLOYMENT PACKAGE FOOTPRINT"
)


package_map = json.loads(
    PACKAGE_MAP.read_text(
        encoding="utf-8"
    )
)


package_rows = []


for target_id, info in sorted(
    package_map[
        "targets"
    ].items()
):

    row = {
        "target_id":
            target_id,

        "serialized_model_size_bytes":
            int(
                info[
                    "serialized_model_size_bytes"
                ]
            ),

        "serialized_model_size_mib":
            float(
                info[
                    "serialized_model_size_bytes"
                ]
            )
            /
            1024**2,

        "deployment_package_size_bytes":
            int(
                info[
                    "deployment_package_size_bytes"
                ]
            ),

        "deployment_package_size_mib":
            float(
                info[
                    "deployment_package_size_bytes"
                ]
            )
            /
            1024**2,

        "serialized_artifact_count":
            len(
                info[
                    "serialized_model_artifacts"
                ]
            ),

        "deployment_artifact_count":
            len(
                info[
                    "deployment_package_artifacts"
                ]
            ),
    }


    package_rows.append(
        row
    )


    print(
        f"{target_id:43s} "
        f"serialized={row['serialized_model_size_mib']:8.3f} MiB "
        f"deployment={row['deployment_package_size_mib']:8.3f} MiB"
    )


with PACKAGE_SIZE_CSV.open(
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=list(
            package_rows[
                0
            ].keys()
        ),
    )

    writer.writeheader()

    writer.writerows(
        package_rows
    )


# =============================================================================
# 14. SUMMARY JSON
# =============================================================================

summary_payload = {
    "schema":
        "stage26_3_memory_summary_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-3B",

    "parent_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "memory_execution_plan_sha256":
        EXPECTED_MEMORY_PLAN_SHA256,

    "deployment_package_map_sha256":
        EXPECTED_PACKAGE_MAP_SHA256,

    "memory_worker_sha256":
        EXPECTED_WORKER_SHA256,

    "planned_memory_observations":
        400,

    "observed_final_receipts":
        400,

    "status_counts":
        status_counts,

    "rss_sampling_interval_ms":
        5,

    "fresh_process_per_repetition":
        True,

    "inference_passes_per_repetition":
        1,

    "timing_performed":
        False,

    "gpu_used":
        False,

    "holdout_reopened":
        False,

    "condition_summaries":
        summary_rows,

    "package_sizes":
        package_rows,

    "claim_boundary":
        (
            "Stage26-3B characterizes CPU process RSS and immutable "
            "deployment-package footprint under the predeclared memory "
            "conditions. It does not measure latency, extraction, "
            "end-to-end performance, GPU memory, capacity, or Pareto status."
        ),
}


atomic_json(
    SUMMARY_JSON,
    summary_payload,
)


# =============================================================================
# 15. EXECUTION RECORD
# =============================================================================

execution_record = {
    "schema":
        "stage26_3b_execution_record_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-3B",

    "completed_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_HEAD,

    "memory_plan_sha256":
        EXPECTED_MEMORY_PLAN_SHA256,

    "memory_worker_sha256":
        EXPECTED_WORKER_SHA256,

    "worker_ram_fraction_limit":
        MAX_WORKER_RAM_FRACTION,

    "worker_ram_limit_bytes":
        worker_ram_limit_bytes,

    "parent_rss_guard_sampling_ms":
        5,

    "worker_internal_rss_sampling_ms":
        5,

    "resource_timeout_seconds":
        CONDITION_TIMEOUT_SECONDS,

    "environment_gate":
        {
            "cpu_percent_max":
                CPU_UTILIZATION_GATE_PERCENT,

            "available_ram_gib_min":
                MIN_AVAILABLE_RAM_GIB,

            "retry_count":
                ENVIRONMENT_RETRY_COUNT,
        },

    "memory_observation_count":
        400,

    "status_counts":
        status_counts,

    "completed_observation_remeasured":
        False,

    "timing_performed":
        False,

    "gpu_used":
        False,

    "holdout_reopened":
        False,
}


atomic_json(
    EXECUTION_RECORD,
    execution_record,
)


# =============================================================================
# 16. FINAL STAGE26-3B RECEIPT
# =============================================================================

overall_status = (
    "PASS"
    if status_counts == {
        "PASS":
            400
    }
    else
    "COMPLETE_WITH_FROZEN_RESOURCE_LIMITS"
)


receipt = {
    "schema":
        "stage26_3b_memory_receipt_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-3B",

    "status":
        overall_status,

    "completed_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_HEAD,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "cpu_execution_plan_sha256":
        EXPECTED_CPU_PLAN_SHA256,

    "stage26_2_receipt_sha256":
        EXPECTED_STAGE26_2_RECEIPT_SHA256,

    "memory_execution_plan_sha256":
        EXPECTED_MEMORY_PLAN_SHA256,

    "deployment_package_map_sha256":
        EXPECTED_PACKAGE_MAP_SHA256,

    "memory_worker_sha256":
        EXPECTED_WORKER_SHA256,

    "memory_implementation_sha256":
        EXPECTED_IMPLEMENTATION_SHA256,

    "freeze_receipt_sha256":
        EXPECTED_FREEZE_RECEIPT_SHA256,

    "planned_memory_observations":
        400,

    "final_memory_observations":
        400,

    "pass_observations":
        int(
            status_counts.get(
                "PASS",
                0,
            )
        ),

    "resource_limit_oom_observations":
        int(
            status_counts.get(
                "RESOURCE_LIMIT_OOM",
                0,
            )
        ),

    "timeout_resource_limit_observations":
        int(
            status_counts.get(
                "TIMEOUT_RESOURCE_LIMIT",
                0,
            )
        ),

    "raw_jsonl_sha256":
        sha256_file(
            RAW_JSONL
        ),

    "raw_csv_sha256":
        sha256_file(
            RAW_CSV
        ),

    "summary_csv_sha256":
        sha256_file(
            SUMMARY_CSV
        ),

    "summary_json_sha256":
        sha256_file(
            SUMMARY_JSON
        ),

    "package_size_csv_sha256":
        sha256_file(
            PACKAGE_SIZE_CSV
        ),

    "execution_record_sha256":
        sha256_file(
            EXECUTION_RECORD
        ),

    "memory_profiled":
        True,

    "package_footprint_profiled":
        True,

    "performance_timing_performed":
        False,

    "feature_extraction_measured":
        False,

    "end_to_end_pipeline_measured":
        False,

    "gpu_used":
        False,

    "holdout_reopened":
        False,

    "pareto_analysis_performed":
        False,

    "next_action":
        "GIT_ANCHOR_STAGE26_3B_BEFORE_EXTRACTION_PROFILING",
}


atomic_json(
    RECEIPT_PATH,
    receipt,
)


receipt_sha = sha256_file(
    RECEIPT_PATH
)


# =============================================================================
# 17. DISPLAY COMPACT MEMORY RESULTS
# =============================================================================

banner(
    "STAGE26-3B :: MEDIAN MEMORY SUMMARY"
)


for target_id in sorted(
    {
        row[
            "target_id"
        ]
        for row in summary_rows
    }
):

    print(
        "\n"
        +
        target_id
    )


    target_rows = [
        row
        for row in summary_rows
        if row[
            "target_id"
        ] == target_id
    ]


    target_rows = sorted(
        target_rows,
        key=lambda row: (
            row[
                "hardware_mode"
            ],
            int(
                row[
                    "batch_size"
                ]
            ),
        ),
    )


    for row in target_rows:

        if row[
            "pass_repetitions"
        ]:

            print(
                f"  {row['hardware_mode']:21s} "
                f"B={int(row['batch_size']):5d} "
                f"PASS={row['pass_repetitions']}/5 "
                f"median_loaded="
                f"{mib(row['median_loaded_rss_bytes']):9.2f} MiB "
                f"median_peak="
                f"{mib(row['median_peak_rss_bytes']):10.2f} MiB "
                f"median_Δpeak="
                f"{mib(row['median_delta_peak_rss_bytes']):10.2f} MiB"
            )

        else:

            print(
                f"  {row['hardware_mode']:21s} "
                f"B={int(row['batch_size']):5d} "
                f"PASS=0/5 "
                f"OOM={row['resource_limit_oom_repetitions']} "
                f"TIMEOUT={row['timeout_resource_limit_repetitions']}"
            )


# =============================================================================
# 18. BUILD DURABLE REPOSITORY PACKAGE
# =============================================================================

banner(
    "STAGE26-3B :: BUILD DURABLE RESULT PACKAGE"
)


if RESULT_PACKAGE_DIR.exists():

    raise RuntimeError(
        "Stage26-3B result package unexpectedly already exists."
    )


RESULT_PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


for source in [
    RAW_JSONL,
    RAW_CSV,
    SUMMARY_CSV,
    SUMMARY_JSON,
    PACKAGE_SIZE_CSV,
    EXECUTION_RECORD,
    RECEIPT_PATH,
]:

    shutil.copy2(
        source,
        RESULT_PACKAGE_DIR
        / source.name,
    )


repo_obs_dir = (
    RESULT_PACKAGE_DIR
    / "observations"
)


repo_obs_dir.mkdir(
    parents=True,
    exist_ok=False,
)


for path in observation_paths:

    shutil.copy2(
        path,
        repo_obs_dir
        / path.name,
    )


print(
    "[COPIED] 400 individual memory observation receipts"
)


# Preserve exact frozen lock identities by reference, not duplicate bytes.
LOCK_REFERENCE_PATH = (
    RESULT_PACKAGE_DIR
    / "stage26_3a_lock_reference.json"
)


atomic_json(
    LOCK_REFERENCE_PATH,
    {
        "schema":
            "stage26_3a_lock_reference_v1",

        "parent_memory_lock_commit":
            EXPECTED_HEAD,

        "measurement_protocol_sha256":
            EXPECTED_PROTOCOL_SHA256,

        "memory_execution_plan_sha256":
            EXPECTED_MEMORY_PLAN_SHA256,

        "deployment_package_map_sha256":
            EXPECTED_PACKAGE_MAP_SHA256,

        "memory_worker_sha256":
            EXPECTED_WORKER_SHA256,

        "memory_implementation_sha256":
            EXPECTED_IMPLEMENTATION_SHA256,

        "freeze_receipt_sha256":
            EXPECTED_FREEZE_RECEIPT_SHA256,
    },
)


# =============================================================================
# 19. PACKAGE MANIFEST
# =============================================================================

PACKAGE_MANIFEST = (
    RESULT_PACKAGE_DIR
    / "stage26_3b_memory_package_manifest.json"
)


manifest_files = []


for path in sorted(
    RESULT_PACKAGE_DIR.rglob(
        "*"
    )
):

    if (
        path.is_file()
        and
        path != PACKAGE_MANIFEST
    ):

        manifest_files.append(
            {
                "path":
                    str(
                        path.relative_to(
                            REPO
                        )
                    ),

                "size_bytes":
                    int(
                        path.stat().st_size
                    ),

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )


atomic_json(
    PACKAGE_MANIFEST,
    {
        "schema":
            "stage26_3b_memory_package_manifest_v1",

        "stage":
            26,

        "checkpoint":
            "STAGE26-3B",

        "status":
            "READY_FOR_GIT_ANCHOR",

        "parent_commit":
            EXPECTED_HEAD,

        "receipt_sha256":
            receipt_sha,

        "memory_observation_receipt_count":
            400,

        "file_count_excluding_manifest":
            len(
                manifest_files
            ),

        "files":
            manifest_files,

        "scientific_boundary":
            {
                "memory_profiled":
                    True,

                "package_footprint_profiled":
                    True,

                "performance_timing_performed":
                    False,

                "feature_extraction_measured":
                    False,

                "end_to_end_measured":
                    False,

                "gpu_used":
                    False,

                "holdout_reopened":
                    False,
            },
    },
)


manifest_sha = sha256_file(
    PACKAGE_MANIFEST
)


# =============================================================================
# 20. FINAL UPSTREAM IMMUTABILITY + REPO AUDIT
# =============================================================================

banner(
    "STAGE26-3B :: FINAL AUDIT"
)


for name, path, expected_sha in immutable_checks:

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:

        raise RuntimeError(
            f"Frozen upstream artifact changed during memory profiling: {name}"
        )


repo_status_after = git(
    "status",
    "--porcelain",
)


print(
    "Repository status:"
)

print(
    repo_status_after
    if repo_status_after
    else
    "<unexpected clean>"
)


if not repo_status_after:

    raise RuntimeError(
        "Expected uncommitted Stage26-3B result package."
    )


unexpected = []


for line in repo_status_after.splitlines():

    relpath = line[
        3:
    ]


    if not relpath.startswith(
        "results/stage26_deployment_profiling/"
        "stage26_3b_cpu_memory_profile/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository modifications outside Stage26-3B:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 21. CLOSURE
# =============================================================================

banner(
    "STAGE26-3B CPU MEMORY PROFILING COMPLETE"
)


print(
    "PARENT:"
)

print(
    " ",
    EXPECTED_HEAD,
)


print(
    "\nMEMORY OBSERVATIONS:"
)

print(
    "  FINAL RECEIPTS            : 400"
)

print(
    "  PASS                      :",
    status_counts.get(
        "PASS",
        0,
    ),
)

print(
    "  RESOURCE_LIMIT_OOM        :",
    status_counts.get(
        "RESOURCE_LIMIT_OOM",
        0,
    ),
)

print(
    "  TIMEOUT_RESOURCE_LIMIT    :",
    status_counts.get(
        "TIMEOUT_RESOURCE_LIMIT",
        0,
    ),
)


print(
    "\nRAW JSONL SHA256:"
)

print(
    " ",
    sha256_file(
        RAW_JSONL
    ),
)


print(
    "\nRAW CSV SHA256:"
)

print(
    " ",
    sha256_file(
        RAW_CSV
    ),
)


print(
    "\nSUMMARY CSV SHA256:"
)

print(
    " ",
    sha256_file(
        SUMMARY_CSV
    ),
)


print(
    "\nSUMMARY JSON SHA256:"
)

print(
    " ",
    sha256_file(
        SUMMARY_JSON
    ),
)


print(
    "\nPACKAGE SIZE CSV SHA256:"
)

print(
    " ",
    sha256_file(
        PACKAGE_SIZE_CSV
    ),
)


print(
    "\nEXECUTION RECORD SHA256:"
)

print(
    " ",
    sha256_file(
        EXECUTION_RECORD
    ),
)


print(
    "\nRECEIPT SHA256:"
)

print(
    " ",
    receipt_sha,
)


print(
    "\nPACKAGE MANIFEST SHA256:"
)

print(
    " ",
    manifest_sha,
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  CPU MEMORY PROFILE COMPLETE"
)

print(
    "  400 / 400 FROZEN MEMORY REPETITIONS HAVE FINAL RECEIPTS"
)

print(
    "  FRESH PROCESS USED FOR EVERY REPETITION"
)

print(
    "  5 ms RSS SAMPLING USED"
)

print(
    "  BASELINE / LOADED / PEAK RSS RECORDED"
)

print(
    "  DELTA MODEL / DELTA PEAK RSS RECORDED"
)

print(
    "  ru_maxrss RETAINED DESCRIPTIVELY"
)

print(
    "  SERIALIZED MODEL FOOTPRINT RECORDED"
)

print(
    "  DEPLOYMENT PACKAGE FOOTPRINT RECORDED"
)

print(
    "  PERFORMANCE TIMING NOT PERFORMED"
)

print(
    "  FEATURE EXTRACTION NOT YET MEASURED"
)

print(
    "  END-TO-END NOT YET MEASURED"
)

print(
    "  PARETO NOT YET PERFORMED"
)

print(
    "  GPU NOT USED"
)

print(
    "  HOLDOUT NOT REOPENED"
)


print(
    "\nCRITICAL NEXT ACTION:"
)

print(
    "  STAGE26-3B-GIT — commit, push, and remotely verify the complete"
)

print(
    "  CPU memory/package results BEFORE Stage26-4 extraction profiling."
)


STAGE26-3B :: PARENT / IMMUTABILITY GATE
Expected HEAD : 2e1ece6cc198b054b01e467275951caafb6af794
Local HEAD    : 2e1ece6cc198b054b01e467275951caafb6af794
origin/main   : 2e1ece6cc198b054b01e467275951caafb6af794
Repository clean: True
measurement protocol         PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
CPU plan                     PASS b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363
Stage26-2 receipt            PASS b4b2623eabde7dd6b9acc250357a9f1ad61fa342c1dd496dcb634d4bfaca4d15
memory plan                  PASS 018584b8f8c94d6afa5fcf3576c261bae2b0f694c8b748bf2e0421234e17ea13
package map                  PASS 09e0d093a8ca83f4d68e9affd675af4336b982a6af14e5034fc2e5ffd4bc547c
memory worker                PASS 2ba87c88244b8b3d8b0265d44b29204dc87e321e5aedc35dc5ac536de5fa8fd7
memory implementation        PASS 9e8439d41924f5bdd13dc4db0a8e997fb9a6be1883a38a9ff4028c9f785452dc
freeze receipt               PASS 465cc0c89f09c42f0d33b4dc1f8e759983fcc

In [24]:
# =============================================================================
# STAGE26-3B-GIT — SAFE QUEUED COMMIT / PUSH / REMOTE VERIFICATION
#
# Queue this cell AFTER the currently-running Stage26-3B memory cell.
#
# SAFETY:
#   - refuses partial Stage26-3B results
#   - requires exactly 400 final memory receipts
#   - verifies receipt counts/statuses
#   - verifies every manifest-listed file SHA256 + size
#   - verifies aggregate-result hashes against final receipt
#   - verifies Stage26-3A + upstream immutable locks
#   - stages ONLY Stage26-3B result directory
#   - commits
#   - pushes to origin/main
#   - fetches back and byte-verifies remote package
#   - no inference
#   - no memory measurement
#   - no timing
#   - no GPU
#   - no holdout
#
# IDEMPOTENT:
#   If the exact Stage26-3B anchor commit already exists, this becomes
#   verification-only and does not create another commit.
# =============================================================================

from __future__ import annotations

import os
import json
import stat
import hashlib
import tempfile
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient


# =============================================================================
# 0. FROZEN PARENT / PATHS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "2e1ece6cc198b054b01e467275951caafb6af794"
)

COMMIT_MESSAGE = (
    "stage26: anchor CPU memory profiling"
)

RESULT_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_3b_cpu_memory_profile"
)

RESULT_DIR = (
    REPO
    / RESULT_REL
)

MANIFEST_REL = (
    RESULT_REL
    + "/stage26_3b_memory_package_manifest.json"
)

RECEIPT_REL = (
    RESULT_REL
    + "/stage26_3b_memory_receipt.json"
)

RAW_JSONL_REL = (
    RESULT_REL
    + "/stage26_3_memory_raw.jsonl"
)

RAW_CSV_REL = (
    RESULT_REL
    + "/stage26_3_memory_raw.csv"
)

SUMMARY_CSV_REL = (
    RESULT_REL
    + "/stage26_3_memory_summary.csv"
)

SUMMARY_JSON_REL = (
    RESULT_REL
    + "/stage26_3_memory_summary.json"
)

PACKAGE_SIZE_CSV_REL = (
    RESULT_REL
    + "/stage26_3_package_sizes.csv"
)

EXECUTION_RECORD_REL = (
    RESULT_REL
    + "/stage26_3b_execution_record.json"
)

OBS_REL = (
    RESULT_REL
    + "/observations"
)


# =============================================================================
# 1. UPSTREAM IMMUTABLE HASHES
# =============================================================================

PROTOCOL_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/"
    "measurement_protocol.json"
)

CPU_PLAN_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_0_protocol_lock/"
    "cpu_execution_plan.json"
)

STAGE26_2_RECEIPT_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_2_cpu_warm_inference/"
    "stage26_2_warm_cpu_receipt.json"
)

MEMORY_PLAN_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_3a_memory_implementation_lock/"
    "stage26_3_memory_execution_plan.json"
)

PACKAGE_MAP_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_3a_memory_implementation_lock/"
    "stage26_3_deployment_package_map.json"
)

MEMORY_WORKER_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_3a_memory_implementation_lock/"
    "stage26_memory_worker.py"
)

MEMORY_IMPLEMENTATION_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_3a_memory_implementation_lock/"
    "stage26_3_memory_implementation.json"
)

FREEZE_RECEIPT_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_3a_memory_implementation_lock/"
    "stage26_3a_memory_freeze_receipt.json"
)

LOCK_MANIFEST_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_3a_memory_implementation_lock/"
    "stage26_3a_memory_lock_package_manifest.json"
)


UPSTREAM_HASHES = {
    PROTOCOL_REL:
        "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625",

    CPU_PLAN_REL:
        "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363",

    STAGE26_2_RECEIPT_REL:
        "b4b2623eabde7dd6b9acc250357a9f1ad61fa342c1dd496dcb634d4bfaca4d15",

    MEMORY_PLAN_REL:
        "018584b8f8c94d6afa5fcf3576c261bae2b0f694c8b748bf2e0421234e17ea13",

    PACKAGE_MAP_REL:
        "09e0d093a8ca83f4d68e9affd675af4336b982a6af14e5034fc2e5ffd4bc547c",

    MEMORY_WORKER_REL:
        "2ba87c88244b8b3d8b0265d44b29204dc87e321e5aedc35dc5ac536de5fa8fd7",

    MEMORY_IMPLEMENTATION_REL:
        "9e8439d41924f5bdd13dc4db0a8e997fb9a6be1883a38a9ff4028c9f785452dc",

    FREEZE_RECEIPT_REL:
        "465cc0c89f09c42f0d33b4dc1f8e759983fccd2897cef0207c85e777cdee3a19",

    LOCK_MANIFEST_REL:
        "de1b2731a276d4628360478f704ac16feabcad97925f3f1079657f4aced24eb1",
}


# =============================================================================
# 2. HELPERS
# =============================================================================

def banner(text):
    print("\n" + "=" * 112)
    print(text)
    print("=" * 112)


def run(cmd, *, env=None, check=True):

    p = subprocess.run(
        cmd,
        cwd=REPO,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "$ "
            + " ".join(cmd)
            + "\n\n"
            + p.stdout
        )

    return p


def git(*args, env=None, check=True):

    return run(
        ["git", *args],
        env=env,
        check=check,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def git_blob_bytes(ref, relpath):

    p = subprocess.run(
        [
            "git",
            "show",
            f"{ref}:{relpath}",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stderr.decode(
                "utf-8",
                errors="replace",
            )
        )

    return p.stdout


# =============================================================================
# 3. CURRENT GIT / PARENT GATE
# =============================================================================

banner(
    "STAGE26-3B-GIT :: PARENT GATE"
)


HEAD_BEFORE = git(
    "rev-parse",
    "HEAD",
)

REMOTE_BEFORE = git(
    "rev-parse",
    "origin/main",
)


print(
    "Expected parent:",
    EXPECTED_PARENT,
)

print(
    "Local HEAD     :",
    HEAD_BEFORE,
)

print(
    "origin/main    :",
    REMOTE_BEFORE,
)


already_committed = False


if HEAD_BEFORE == EXPECTED_PARENT:

    if REMOTE_BEFORE != EXPECTED_PARENT:

        raise RuntimeError(
            "origin/main changed before Stage26-3B anchor. "
            "Do NOT force push."
        )


else:

    parent = git(
        "rev-parse",
        "HEAD^",
        check=False,
    )

    subject = git(
        "log",
        "-1",
        "--pretty=%s",
    )


    if (
        parent == EXPECTED_PARENT
        and
        subject == COMMIT_MESSAGE
    ):

        already_committed = True

        print(
            "[INFO] Stage26-3B anchor already exists locally."
        )


    else:

        raise RuntimeError(
            "Unexpected HEAD. Refusing to anchor Stage26-3B."
        )


# =============================================================================
# 4. UPSTREAM IMMUTABILITY
# =============================================================================

banner(
    "STAGE26-3B-GIT :: UPSTREAM IMMUTABILITY"
)


for relpath, expected in UPSTREAM_HASHES.items():

    path = (
        REPO
        / relpath
    )

    if not path.exists():
        raise FileNotFoundError(path)


    actual = sha256_file(path)

    ok = (
        actual == expected
    )


    print(
        f"{Path(relpath).name:45s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual}"
    )


    if not ok:

        raise RuntimeError(
            "Upstream frozen artifact changed: "
            + relpath
        )


# =============================================================================
# 5. REQUIRE COMPLETE STAGE26-3B PACKAGE
# =============================================================================

banner(
    "STAGE26-3B-GIT :: COMPLETENESS GATE"
)


if not RESULT_DIR.exists():

    raise RuntimeError(
        "\nStage26-3B result package does not exist.\n"
        "The memory profiling cell has not completed.\n"
        "NO COMMIT WAS ATTEMPTED."
    )


required_paths = [
    MANIFEST_REL,
    RECEIPT_REL,
    RAW_JSONL_REL,
    RAW_CSV_REL,
    SUMMARY_CSV_REL,
    SUMMARY_JSON_REL,
    PACKAGE_SIZE_CSV_REL,
    EXECUTION_RECORD_REL,
]


for relpath in required_paths:

    path = (
        REPO
        / relpath
    )

    if not path.exists():

        raise RuntimeError(
            "Incomplete Stage26-3B package; missing:\n"
            + relpath
        )


obs_dir = (
    REPO
    / OBS_REL
)


observation_paths = sorted(
    obs_dir.glob(
        "memory_*.json"
    )
)


print(
    "Individual observation receipts:",
    len(observation_paths),
)


if len(observation_paths) != 400:

    raise RuntimeError(
        f"Expected 400 observation receipts; "
        f"found {len(observation_paths)}. "
        "Refusing partial commit."
    )


expected_names = {
    f"memory_{i:03d}.json"
    for i in range(
        1,
        401,
    )
}


actual_names = {
    p.name
    for p in observation_paths
}


if actual_names != expected_names:

    raise RuntimeError(
        "Observation receipt filenames are not exactly memory_001..memory_400."
    )


# =============================================================================
# 6. OBSERVATION RECEIPT AUDIT
# =============================================================================

banner(
    "STAGE26-3B-GIT :: 400-OBSERVATION AUDIT"
)


allowed_statuses = {
    "PASS",
    "RESOURCE_LIMIT_OOM",
    "TIMEOUT_RESOURCE_LIMIT",
}


status_counts = {}


for expected_order, path in enumerate(
    observation_paths,
    start=1,
):

    row = json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


    order = int(
        row[
            "memory_execution_order"
        ]
    )


    if order != expected_order:

        raise RuntimeError(
            f"Observation order mismatch in {path.name}."
        )


    status = row[
        "status"
    ]


    if status not in allowed_statuses:

        raise RuntimeError(
            f"Invalid final status in {path.name}: {status}"
        )


    status_counts[
        status
    ] = (
        status_counts.get(
            status,
            0,
        )
        +
        1
    )


    if status == "PASS":

        for key in [
            "baseline_rss_bytes",
            "loaded_rss_bytes",
            "peak_rss_bytes",
            "delta_model_rss_bytes",
            "delta_peak_rss_bytes",
            "ru_maxrss_bytes_descriptive",
        ]:

            if key not in row:

                raise RuntimeError(
                    f"{path.name} missing PASS metric {key}."
                )


        if row.get(
            "timing_performed"
        ) is not False:

            raise RuntimeError(
                f"{path.name} unexpectedly performed timing."
            )


        if row.get(
            "gpu_used"
        ) is not False:

            raise RuntimeError(
                f"{path.name} unexpectedly used GPU."
            )


        if row.get(
            "holdout_accessed"
        ) is not False:

            raise RuntimeError(
                f"{path.name} unexpectedly accessed holdout."
            )


print(
    "PASS                   :",
    status_counts.get(
        "PASS",
        0,
    )
)

print(
    "RESOURCE_LIMIT_OOM     :",
    status_counts.get(
        "RESOURCE_LIMIT_OOM",
        0,
    )
)

print(
    "TIMEOUT_RESOURCE_LIMIT :",
    status_counts.get(
        "TIMEOUT_RESOURCE_LIMIT",
        0,
    )
)

print(
    "TOTAL                  :",
    sum(
        status_counts.values()
    )
)


if sum(
    status_counts.values()
) != 400:

    raise RuntimeError(
        "Final memory status counts do not sum to 400."
    )


# =============================================================================
# 7. FINAL RECEIPT CONSISTENCY
# =============================================================================

banner(
    "STAGE26-3B-GIT :: FINAL RECEIPT GATE"
)


receipt_path = (
    REPO
    / RECEIPT_REL
)

receipt = json.loads(
    receipt_path.read_text(
        encoding="utf-8"
    )
)


if receipt.get(
    "status"
) not in {
    "PASS",
    "COMPLETE_WITH_FROZEN_RESOURCE_LIMITS",
}:

    raise RuntimeError(
        "Stage26-3B final receipt does not contain a valid completion status."
    )


expected_receipt_values = {
    "parent_commit":
        EXPECTED_PARENT,

    "planned_memory_observations":
        400,

    "final_memory_observations":
        400,

    "pass_observations":
        status_counts.get(
            "PASS",
            0,
        ),

    "resource_limit_oom_observations":
        status_counts.get(
            "RESOURCE_LIMIT_OOM",
            0,
        ),

    "timeout_resource_limit_observations":
        status_counts.get(
            "TIMEOUT_RESOURCE_LIMIT",
            0,
        ),

    "memory_profiled":
        True,

    "package_footprint_profiled":
        True,

    "performance_timing_performed":
        False,

    "feature_extraction_measured":
        False,

    "end_to_end_pipeline_measured":
        False,

    "gpu_used":
        False,

    "holdout_reopened":
        False,

    "pareto_analysis_performed":
        False,
}


for key, expected in expected_receipt_values.items():

    actual = receipt.get(
        key
    )


    print(
        f"{key:46s}: {actual!r}"
    )


    if actual != expected:

        raise RuntimeError(
            f"Final receipt mismatch: "
            f"{key}={actual!r}; expected={expected!r}"
        )


# =============================================================================
# 8. AGGREGATE FILE HASHES AGAINST RECEIPT
# =============================================================================

banner(
    "STAGE26-3B-GIT :: AGGREGATE HASH GATE"
)


aggregate_checks = [
    (
        RAW_JSONL_REL,
        "raw_jsonl_sha256",
    ),
    (
        RAW_CSV_REL,
        "raw_csv_sha256",
    ),
    (
        SUMMARY_CSV_REL,
        "summary_csv_sha256",
    ),
    (
        SUMMARY_JSON_REL,
        "summary_json_sha256",
    ),
    (
        PACKAGE_SIZE_CSV_REL,
        "package_size_csv_sha256",
    ),
    (
        EXECUTION_RECORD_REL,
        "execution_record_sha256",
    ),
]


for relpath, receipt_key in aggregate_checks:

    actual = sha256_file(
        REPO
        / relpath
    )

    expected = receipt[
        receipt_key
    ]


    ok = (
        actual == expected
    )


    print(
        f"{Path(relpath).name:40s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual}"
    )


    if not ok:

        raise RuntimeError(
            f"Aggregate result hash mismatch: {relpath}"
        )


RECEIPT_SHA = sha256_file(
    receipt_path
)


# =============================================================================
# 9. PACKAGE MANIFEST GATE
# =============================================================================

banner(
    "STAGE26-3B-GIT :: PACKAGE MANIFEST GATE"
)


manifest_path = (
    REPO
    / MANIFEST_REL
)


manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)


if manifest.get(
    "status"
) != "READY_FOR_GIT_ANCHOR":

    raise RuntimeError(
        "Stage26-3B manifest is not READY_FOR_GIT_ANCHOR."
    )


if manifest.get(
    "parent_commit"
) != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-3B manifest parent mismatch."
    )


if int(
    manifest.get(
        "memory_observation_receipt_count",
        -1,
    )
) != 400:

    raise RuntimeError(
        "Manifest does not preserve exactly 400 observations."
    )


if manifest.get(
    "receipt_sha256"
) != RECEIPT_SHA:

    raise RuntimeError(
        "Manifest receipt hash does not match final receipt."
    )


manifest_files = manifest[
    "files"
]


print(
    "Manifest-listed files:",
    len(
        manifest_files
    ),
)


for index, record in enumerate(
    manifest_files,
    start=1,
):

    relpath = record[
        "path"
    ]

    path = (
        REPO
        / relpath
    )


    if not path.exists():

        raise FileNotFoundError(
            path
        )


    actual_sha = sha256_file(
        path
    )

    actual_size = int(
        path.stat().st_size
    )


    if (
        actual_sha
        !=
        record[
            "sha256"
        ]
        or
        actual_size
        !=
        int(
            record[
                "size_bytes"
            ]
        )
    ):

        raise RuntimeError(
            "Manifest verification failure:\n"
            + relpath
        )


    if (
        index <= 10
        or
        index % 50 == 0
        or
        index == len(
            manifest_files
        )
    ):

        print(
            f"[{index:03d}/{len(manifest_files):03d}] PASS {relpath}"
        )


MANIFEST_SHA = sha256_file(
    manifest_path
)


print(
    "\nFinal receipt SHA256:"
)

print(
    " ",
    RECEIPT_SHA
)

print(
    "Package manifest SHA256:"
)

print(
    " ",
    MANIFEST_SHA
)


# =============================================================================
# 10. GIT STATE / IDEMPOTENT COMMIT
# =============================================================================

banner(
    "STAGE26-3B-GIT :: GIT STATE"
)


STATUS_BEFORE = git(
    "status",
    "--porcelain",
)


print(
    "Repository clean:",
    STATUS_BEFORE == "",
)


if STATUS_BEFORE:

    print(
        STATUS_BEFORE
    )


if not already_committed:

    if not STATUS_BEFORE:

        raise RuntimeError(
            "Stage26-3B package exists but repository is unexpectedly clean."
        )


    unexpected = []


    for line in STATUS_BEFORE.splitlines():

        relpath = line[
            3:
        ]


        if not relpath.startswith(
            RESULT_REL
            + "/"
        ):

            unexpected.append(
                line
            )


    if unexpected:

        raise RuntimeError(
            "Unexpected repository modifications outside Stage26-3B:\n"
            +
            "\n".join(
                unexpected
            )
        )


    # =========================================================================
    # 11. STAGE + COMMIT
    # =========================================================================

    banner(
        "STAGE26-3B-GIT :: COMMIT"
    )


    git(
        "add",
        "--",
        RESULT_REL,
    )


    staged = git(
        "diff",
        "--cached",
        "--name-status",
    )


    if not staged:

        raise RuntimeError(
            "Nothing staged for Stage26-3B."
        )


    staged_lines = staged.splitlines()


    print(
        "Staged files:",
        len(
            staged_lines
        )
    )


    for line in staged_lines:

        relpath = line.split(
            "\t"
        )[
            -1
        ]


        if not relpath.startswith(
            RESULT_REL
            + "/"
        ):

            raise RuntimeError(
                "Unexpected staged path:\n"
                + line
            )


    if not git(
        "config",
        "--get",
        "user.name",
        check=False,
    ):

        git(
            "config",
            "user.name",
            "themubasshir",
        )


    if not git(
        "config",
        "--get",
        "user.email",
        check=False,
    ):

        git(
            "config",
            "user.email",
            "themubasshir@users.noreply.github.com",
        )


    commit_output = git(
        "commit",
        "-m",
        COMMIT_MESSAGE,
    )


    print(
        commit_output
    )


    STAGE26_3B_HEAD = git(
        "rev-parse",
        "HEAD",
    )


    if git(
        "rev-parse",
        "HEAD^",
    ) != EXPECTED_PARENT:

        raise RuntimeError(
            "Stage26-3B commit parent mismatch."
        )


else:

    STAGE26_3B_HEAD = HEAD_BEFORE


print(
    "\nStage26-3B commit:"
)

print(
    " ",
    STAGE26_3B_HEAD
)


# =============================================================================
# 12. COMMITTED BLOB GATE
# =============================================================================

banner(
    "STAGE26-3B-GIT :: COMMITTED BLOB GATE"
)


committed_manifest_sha = sha256_bytes(
    git_blob_bytes(
        "HEAD",
        MANIFEST_REL,
    )
)


if committed_manifest_sha != MANIFEST_SHA:

    raise RuntimeError(
        "Committed Stage26-3B manifest mismatch."
    )


committed_receipt_sha = sha256_bytes(
    git_blob_bytes(
        "HEAD",
        RECEIPT_REL,
    )
)


if committed_receipt_sha != RECEIPT_SHA:

    raise RuntimeError(
        "Committed Stage26-3B receipt mismatch."
    )


print(
    "manifest PASS",
    committed_manifest_sha
)

print(
    "receipt  PASS",
    committed_receipt_sha
)


# =============================================================================
# 13. REMOTE RELATIONSHIP
# =============================================================================

banner(
    "STAGE26-3B-GIT :: REMOTE RELATIONSHIP"
)


git(
    "fetch",
    "origin",
    "main",
)


REMOTE_CURRENT = git(
    "rev-parse",
    "origin/main",
)


print(
    "Local Stage26-3B HEAD:",
    STAGE26_3B_HEAD,
)

print(
    "origin/main currently:",
    REMOTE_CURRENT,
)


if REMOTE_CURRENT == STAGE26_3B_HEAD:

    relationship = (
        "REMOTE_ALREADY_HAS_STAGE26_3B"
    )


elif REMOTE_CURRENT == EXPECTED_PARENT:

    relationship = (
        "LOCAL_STAGE26_3B_AHEAD_BY_ONE"
    )


else:

    relationship = (
        "UNEXPECTED_REMOTE_DIVERGENCE"
    )


print(
    "Relationship:",
    relationship,
)


if relationship == "UNEXPECTED_REMOTE_DIVERGENCE":

    raise RuntimeError(
        "origin/main diverged. Refusing force push."
    )


# =============================================================================
# 14. PUSH
# =============================================================================

if relationship == "LOCAL_STAGE26_3B_AHEAD_BY_ONE":

    banner(
        "STAGE26-3B-GIT :: PUSH"
    )


    secrets = UserSecretsClient()


    aliases = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "gh_token",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
        "gh_pat",
    ]


    token = None
    token_label = None


    for label in aliases:

        try:

            value = secrets.get_secret(
                label
            )

        except Exception:

            value = None


        if value:

            token = value.strip()
            token_label = label
            break


    if not token:

        raise RuntimeError(
            "No usable GitHub credential found in Kaggle Secrets."
        )


    print(
        f"[FOUND] {token_label} "
        f"({len(token)} characters)"
    )

    print(
        "Token value will not be printed."
    )


    fd, askpass_path = tempfile.mkstemp(
        prefix="stage26_3b_askpass_",
        suffix=".sh",
    )

    os.close(fd)


    askpass = Path(
        askpass_path
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' "x-access-token" ;;
  *Password*) printf '%s\\n' "$GITHUB_TOKEN" ;;
  *)          printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        stat.S_IRUSR
        |
        stat.S_IWUSR
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GITHUB_TOKEN"
    ] = token

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"


    try:

        result = run(
            [
                "git",
                "push",
                "origin",
                "HEAD:main",
            ],
            env=push_env,
        )

        print(
            result.stdout
        )


    finally:

        try:
            askpass.unlink()

        except FileNotFoundError:
            pass

        token = None


# =============================================================================
# 15. REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-3B-GIT :: REMOTE COMMIT GATE"
)


git(
    "fetch",
    "origin",
    "main",
)


LOCAL_HEAD = git(
    "rev-parse",
    "HEAD",
)

REMOTE_HEAD = git(
    "rev-parse",
    "origin/main",
)


ls_remote = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


LS_REMOTE_HEAD = (
    ls_remote.split()[0]
    if ls_remote
    else None
)


print(
    "Local HEAD :",
    LOCAL_HEAD,
)

print(
    "Remote HEAD:",
    REMOTE_HEAD,
)

print(
    "ls-remote :",
    LS_REMOTE_HEAD,
)


if not (
    LOCAL_HEAD
    ==
    REMOTE_HEAD
    ==
    LS_REMOTE_HEAD
    ==
    STAGE26_3B_HEAD
):

    raise RuntimeError(
        "Stage26-3B remote commit verification failed."
    )


# =============================================================================
# 16. REMOTE BYTE-FOR-BYTE PACKAGE VERIFICATION
# =============================================================================

banner(
    "STAGE26-3B-GIT :: REMOTE PACKAGE VERIFICATION"
)


remote_manifest_sha = sha256_bytes(
    git_blob_bytes(
        "origin/main",
        MANIFEST_REL,
    )
)


remote_receipt_sha = sha256_bytes(
    git_blob_bytes(
        "origin/main",
        RECEIPT_REL,
    )
)


if remote_manifest_sha != MANIFEST_SHA:

    raise RuntimeError(
        "Remote manifest SHA mismatch."
    )


if remote_receipt_sha != RECEIPT_SHA:

    raise RuntimeError(
        "Remote receipt SHA mismatch."
    )


verified = 0


for index, record in enumerate(
    manifest_files,
    start=1,
):

    relpath = record[
        "path"
    ]

    remote_sha = sha256_bytes(
        git_blob_bytes(
            "origin/main",
            relpath,
        )
    )


    if remote_sha != record[
        "sha256"
    ]:

        raise RuntimeError(
            "Remote file mismatch:\n"
            + relpath
        )


    verified += 1


    if (
        index <= 10
        or
        index % 50 == 0
        or
        index == len(
            manifest_files
        )
    ):

        print(
            f"[{index:03d}/{len(manifest_files):03d}] PASS"
        )


print(
    "\nRemote manifest-listed files verified:",
    verified,
)


# =============================================================================
# 17. REMOTE UPSTREAM LOCK VERIFICATION
# =============================================================================

banner(
    "STAGE26-3B-GIT :: REMOTE UPSTREAM IMMUTABILITY"
)


for relpath, expected in UPSTREAM_HASHES.items():

    actual = sha256_bytes(
        git_blob_bytes(
            "origin/main",
            relpath,
        )
    )


    if actual != expected:

        raise RuntimeError(
            "Remote upstream lock mismatch:\n"
            + relpath
        )


    print(
        f"PASS {Path(relpath).name}"
    )


# =============================================================================
# 18. FINAL CLEANLINESS
# =============================================================================

banner(
    "STAGE26-3B-GIT :: FINAL AUDIT"
)


FINAL_STATUS = git(
    "status",
    "--porcelain",
)


print(
    "Repository clean:",
    FINAL_STATUS == "",
)


if FINAL_STATUS:

    print(
        FINAL_STATUS
    )

    raise RuntimeError(
        "Repository dirty after Stage26-3B anchor."
    )


# =============================================================================
# 19. CLOSURE
# =============================================================================

banner(
    "STAGE26-3B-GIT COMPLETE"
)


print(
    "PARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nSTAGE26-3B COMMIT:"
)

print(
    " ",
    STAGE26_3B_HEAD
)


print(
    "\nREMOTE MAIN:"
)

print(
    " ",
    REMOTE_HEAD
)


print(
    "\nMEMORY OBSERVATIONS:"
)

print(
    "  PASS                   :",
    status_counts.get(
        "PASS",
        0,
    )
)

print(
    "  RESOURCE_LIMIT_OOM     :",
    status_counts.get(
        "RESOURCE_LIMIT_OOM",
        0,
    )
)

print(
    "  TIMEOUT_RESOURCE_LIMIT :",
    status_counts.get(
        "TIMEOUT_RESOURCE_LIMIT",
        0,
    )
)

print(
    "  TOTAL                  : 400"
)


print(
    "\nFINAL RECEIPT SHA256:"
)

print(
    " ",
    RECEIPT_SHA
)


print(
    "\nPACKAGE MANIFEST SHA256:"
)

print(
    " ",
    MANIFEST_SHA
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  COMMIT MATCH              : PASS"
)

print(
    "  LS-REMOTE                 : PASS"
)

print(
    "  RECEIPT                   : PASS"
)

print(
    "  ALL MANIFEST FILES        : PASS"
)

print(
    "  ALL UPSTREAM LOCKS        : PASS"
)

print(
    "  REPOSITORY CLEAN          : PASS"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  STAGE26-3 CPU MEMORY + PACKAGE FOOTPRINT DURABLY ANCHORED"
)

print(
    "  400 / 400 FINAL MEMORY RECEIPTS PRESERVED"
)

print(
    "  RESOURCE-LIMIT OUTCOMES PRESERVED"
)

print(
    "  NO PERFORMANCE TIMING ADDED"
)

print(
    "  GPU NOT USED"
)

print(
    "  HOLDOUT NOT REOPENED"
)


print(
    "\nNEXT AFTER YOU WAKE UP:"
)

print(
    "  STAGE26-4 — freeze the exact PCAP / extraction protocol"
)

print(
    "  BEFORE performing any extraction benchmark."
)


STAGE26-3B-GIT :: PARENT GATE
Expected parent: 2e1ece6cc198b054b01e467275951caafb6af794
Local HEAD     : 2e1ece6cc198b054b01e467275951caafb6af794
origin/main    : 2e1ece6cc198b054b01e467275951caafb6af794

STAGE26-3B-GIT :: UPSTREAM IMMUTABILITY
measurement_protocol.json                     PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
cpu_execution_plan.json                       PASS b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363
stage26_2_warm_cpu_receipt.json               PASS b4b2623eabde7dd6b9acc250357a9f1ad61fa342c1dd496dcb634d4bfaca4d15
stage26_3_memory_execution_plan.json          PASS 018584b8f8c94d6afa5fcf3576c261bae2b0f694c8b748bf2e0421234e17ea13
stage26_3_deployment_package_map.json         PASS 09e0d093a8ca83f4d68e9affd675af4336b982a6af14e5034fc2e5ffd4bc547c
stage26_memory_worker.py                      PASS 2ba87c88244b8b3d8b0265d44b29204dc87e321e5aedc35dc5ac536de5fa8fd7
stage26_3_memory_implementation.json          PASS 9e8439d

In [2]:
# =============================================================================
# STAGE26 — FRESH KAGGLE SESSION RECOVERY
#
# PURPOSE:
#   Recover after Kaggle session reset from the last remotely durable CPU
#   checkpoint.
#
# REMOTE SCIENTIFIC ANCHOR:
#   347d93f21d454cc5bda2c45889c890c67cdf0ecc
#   "stage26: anchor CPU memory profiling"
#
# THIS CELL DOES NOT:
#   - rerun Stage26-1 cold start
#   - rerun Stage26-2 warm inference
#   - rerun Stage26-3 memory profiling
#   - load any model
#   - perform inference
#   - perform timing
#   - read/parse PCAP data
#   - access labels
#   - access Thursday or Friday
#   - use GPU
#   - modify GitHub
#
# It ONLY:
#   1. restores/verifies the exact Git repository;
#   2. verifies key durable Stage26 hashes;
#   3. recreates empty Stage26 working directories;
#   4. confirms GITHUB_TOKEN exists without printing it;
#   5. records a local reset-recovery receipt.
# =============================================================================

from __future__ import annotations

import os
import json
import hashlib
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. FROZEN ANCHOR
# =============================================================================

REPO_URL = (
    "https://github.com/themubasshir/"
    "ids2018-validation-safe-ablation.git"
)

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "347d93f21d454cc5bda2c45889c890c67cdf0ecc"
)

EXPECTED_COMMIT_SUBJECT = (
    "stage26: anchor CPU memory profiling"
)


# -----------------------------------------------------------------------------
# Known immutable Stage26 hashes
# -----------------------------------------------------------------------------

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

EXPECTED_STAGE26_3B_RECEIPT_SHA256 = (
    "db4b39af6b78299b67a13fad1f218a701894ef0af6d705552146173ee22939ac"
)

EXPECTED_STAGE26_3B_MANIFEST_SHA256 = (
    "ab73d0c3fab3905301acc28afbd01912d8021fe50e1e3b25ff818244eeea0b06"
)


STAGE26_PROTOCOL = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_0_protocol_lock/"
      "measurement_protocol.json"
)

STAGE26_3B_RECEIPT = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_3b_cpu_memory_profile/"
      "stage26_3b_memory_receipt.json"
)

STAGE26_3B_MANIFEST = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_3b_cpu_memory_profile/"
      "stage26_3b_memory_package_manifest.json"
)


# -----------------------------------------------------------------------------
# Recreated volatile workspace
# -----------------------------------------------------------------------------

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

WORKDIRS = [
    "protocol",
    "runtime",
    "raw_timings",
    "cold_start",
    "steady_state",
    "memory",
    "package_size",
    "sources",
    "extraction",
    "pipeline",
    "pareto",
    "figures",
    "tables",
    "logs",
    "workers",
]

RECOVERY_RECEIPT = (
    STAGE26_ROOT
    / "runtime"
    / "stage26_fresh_session_recovery.json"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):
    print("\n" + "=" * 112)
    print(text)
    print("=" * 112)


def run(cmd, *, cwd=None, check=True):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}\n"
            f"{p.stdout}"
        )

    return p.stdout.strip()


def git(*args):
    return run(
        ["git", *args],
        cwd=REPO,
    )


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            chunk = f.read(8 * 1024 * 1024)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


def write_json_atomic(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = Path(str(path) + ".tmp")

    with tmp.open("w", encoding="utf-8") as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )
        f.write("\n")
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)


# =============================================================================
# 2. BASIC FRESH-SESSION AUDIT
# =============================================================================

banner("STAGE26 RESET RECOVERY :: RUNTIME")

print("Python executable :", shutil.which("python") or "unknown")
print("Working directory :", Path("/kaggle/working"))
print("Kaggle input exists:", Path("/kaggle/input").exists())
print("GPU visibility env :", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))

# Stage26 CPU phase remains CPU-only.
os.environ["CUDA_VISIBLE_DEVICES"] = ""

print("GPU hidden for recovery cell: YES")


# =============================================================================
# 3. RESTORE REPOSITORY WITHOUT DESTROYING UNKNOWN FILES
# =============================================================================

banner("STAGE26 RESET RECOVERY :: REPOSITORY RESTORATION")


if REPO.exists():

    if not (REPO / ".git").exists():
        raise RuntimeError(
            f"{REPO} exists but is not a Git repository.\n"
            "Refusing to delete or overwrite an unknown directory."
        )

    print("Existing repository found; auditing instead of recloning.")

    pre_status = git("status", "--porcelain")

    if pre_status:
        raise RuntimeError(
            "Existing repository is not clean. Refusing destructive recovery:\n"
            + pre_status
        )

    git("fetch", "--prune", "origin", "main")

else:

    print("Repository absent after reset.")
    print("Cloning clean main branch...")

    run([
        "git",
        "clone",
        "--branch", "main",
        "--single-branch",
        REPO_URL,
        str(REPO),
    ])


# =============================================================================
# 4. EXACT REMOTE/LOCAL ANCHOR GATE
# =============================================================================

banner("STAGE26 RESET RECOVERY :: SCIENTIFIC GIT ANCHOR")


git("fetch", "origin", "main")

local_head = git("rev-parse", "HEAD")
remote_head = git("rev-parse", "origin/main")
subject = git("show", "-s", "--format=%s", "HEAD")
status = git("status", "--porcelain")


print("Expected HEAD :", EXPECTED_HEAD)
print("Local HEAD    :", local_head)
print("origin/main   :", remote_head)
print("Commit subject:", subject)
print("Repo clean    :", status == "")


if remote_head != EXPECTED_HEAD:
    raise RuntimeError(
        "origin/main is not the frozen Stage26-3B anchor.\n"
        "STOP: do not continue Stage26 until this is investigated."
    )


# If clone/default state somehow differs but worktree is clean,
# move local main to exactly the remotely verified frozen commit.
if local_head != EXPECTED_HEAD:

    if status:
        raise RuntimeError(
            "Local HEAD differs and worktree is not clean."
        )

    print("\nLocal HEAD differs from verified remote anchor.")
    print("Resetting CLEAN local checkout to exact origin/main...")

    git("reset", "--hard", EXPECTED_HEAD)

    local_head = git("rev-parse", "HEAD")
    status = git("status", "--porcelain")


if local_head != EXPECTED_HEAD:
    raise RuntimeError(
        "Could not anchor local checkout to expected Stage26-3B commit."
    )


if subject != EXPECTED_COMMIT_SUBJECT:
    # Re-read in case a reset occurred above.
    subject = git("show", "-s", "--format=%s", "HEAD")

    if subject != EXPECTED_COMMIT_SUBJECT:
        raise RuntimeError(
            "Commit subject does not match expected Stage26-3B anchor."
        )


if status:
    raise RuntimeError(
        "Repository is not clean after recovery."
    )


print("\n[PASS] Exact Stage26-3B Git anchor restored.")


# =============================================================================
# 5. VERIFY DURABLE SCIENTIFIC ARTIFACTS
# =============================================================================

banner("STAGE26 RESET RECOVERY :: DURABLE ARTIFACT HASH GATE")


checks = [
    (
        "Stage26 measurement protocol",
        STAGE26_PROTOCOL,
        EXPECTED_PROTOCOL_SHA256,
    ),
    (
        "Stage26-3B memory receipt",
        STAGE26_3B_RECEIPT,
        EXPECTED_STAGE26_3B_RECEIPT_SHA256,
    ),
    (
        "Stage26-3B package manifest",
        STAGE26_3B_MANIFEST,
        EXPECTED_STAGE26_3B_MANIFEST_SHA256,
    ),
]


hash_results = {}


for name, path, expected_sha in checks:

    if not path.exists():
        raise FileNotFoundError(
            f"Missing durable artifact:\n{path}"
        )

    actual_sha = sha256_file(path)

    ok = actual_sha == expected_sha

    hash_results[name] = {
        "path": str(path),
        "expected_sha256": expected_sha,
        "actual_sha256": actual_sha,
        "pass": ok,
    }

    print(
        f"{name:34s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual_sha}"
    )

    if not ok:
        raise RuntimeError(
            f"Durable artifact hash mismatch: {name}"
        )


print("\n[PASS] Stage26 completed CPU science survived session reset.")


# =============================================================================
# 6. READ STAGE26-3B RECEIPT — NO MODEL WORK
# =============================================================================

banner("STAGE26 RESET RECOVERY :: STAGE26-3B STATE")


memory_receipt = json.loads(
    STAGE26_3B_RECEIPT.read_text(encoding="utf-8")
)


# Print only high-level metadata already committed.
for key in (
    "status",
    "completed_conditions",
    "total_conditions",
    "pass_count",
    "oom_count",
    "timeout_count",
):
    if key in memory_receipt:
        print(f"{key:24s}: {memory_receipt[key]}")


print(
    "\nNote: Stage26-1/2/3 are REMOTELY DURABLE and will NOT be rerun."
)


# =============================================================================
# 7. RECREATE ONLY EMPTY VOLATILE STAGE26 WORKSPACE
# =============================================================================

banner("STAGE26 RESET RECOVERY :: VOLATILE WORKSPACE")


STAGE26_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


for name in WORKDIRS:
    path = STAGE26_ROOT / name
    path.mkdir(
        parents=True,
        exist_ok=True,
    )
    print("[READY]", path)


# =============================================================================
# 8. VERIFY GITHUB TOKEN SECRET WITHOUT EXPOSING IT
# =============================================================================

banner("STAGE26 RESET RECOVERY :: GITHUB SECRET")


github_secret_present = False
github_secret_length = None


try:
    from kaggle_secrets import UserSecretsClient

    secret_client = UserSecretsClient()

    token = secret_client.get_secret(
        "GITHUB_TOKEN"
    )

    if token:
        github_secret_present = True
        github_secret_length = len(token)

        print(
            f"[FOUND] GITHUB_TOKEN "
            f"({github_secret_length} characters)"
        )

    else:
        print("[WARN] GITHUB_TOKEN returned an empty value.")

    # Do not retain a second visible representation.
    del token

except Exception as exc:
    print(
        "[WARN] GITHUB_TOKEN could not be retrieved in this session:"
    )
    print(
        " ",
        type(exc).__name__,
        str(exc),
    )


# Token absence is NOT a scientific failure and does not invalidate
# the restored repository. It only matters when a later push is needed.


# =============================================================================
# 9. RECORD LOCAL RESET-RECOVERY RECEIPT
# =============================================================================

recovery = {
    "schema":
        "stage26_fresh_session_recovery_v1",

    "created_at_utc":
        datetime.now(timezone.utc).isoformat(),

    "reason":
        "KAGGLE_SESSION_RESET",

    "scientific_anchor": {
        "expected_head":
            EXPECTED_HEAD,

        "local_head":
            local_head,

        "origin_main":
            remote_head,

        "commit_subject":
            subject,

        "repository_clean":
            status == "",
    },

    "durable_hash_gate":
        hash_results,

    "volatile_workspace": {
        "root":
            str(STAGE26_ROOT),

        "recreated_directories":
            WORKDIRS,
    },

    "github_secret": {
        "GITHUB_TOKEN_present":
            github_secret_present,

        "length_if_present":
            github_secret_length,

        "token_value_recorded":
            False,
    },

    "scientific_boundary": {
        "stage26_1_rerun":
            False,

        "stage26_2_rerun":
            False,

        "stage26_3_rerun":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "memory_measurement_performed":
            False,

        "pcap_opened":
            False,

        "pcap_parsed":
            False,

        "labels_read":
            False,

        "Thursday_accessed":
            False,

        "Friday_accessed":
            False,

        "gpu_used":
            False,

        "github_modified":
            False,
    },

    "next_checkpoint":
        "STAGE26-4A_EXACT_MONDAY_PCAP_RESTORATION",
}


write_json_atomic(
    RECOVERY_RECEIPT,
    recovery,
)


recovery_sha = sha256_file(
    RECOVERY_RECEIPT
)


# =============================================================================
# 10. FINAL AUDIT
# =============================================================================

banner("STAGE26 FRESH-SESSION RECOVERY COMPLETE")


final_head = git("rev-parse", "HEAD")
final_remote = git("rev-parse", "origin/main")
final_status = git("status", "--porcelain")


print("Local HEAD       :", final_head)
print("origin/main      :", final_remote)
print("Repository clean :", final_status == "")

print("\nRecovery receipt :")
print(" ", RECOVERY_RECEIPT)
print(" SHA256:", recovery_sha)


if final_head != EXPECTED_HEAD:
    raise RuntimeError(
        "Final local HEAD changed unexpectedly."
    )

if final_remote != EXPECTED_HEAD:
    raise RuntimeError(
        "Final origin/main changed unexpectedly."
    )

if final_status:
    raise RuntimeError(
        "Repository became dirty during recovery."
    )


print("\nSCIENTIFIC STATE")
print("  Stage26-1 cold-start measurements preserved : YES")
print("  Stage26-2 warm CPU measurements preserved   : YES")
print("  Stage26-3 memory measurements preserved     : YES")
print("  Completed CPU work rerun                     : NO")
print("  New inference                                : NO")
print("  New timing                                   : NO")
print("  PCAP accessed                                : NO")
print("  Holdout accessed                             : NO")
print("  GPU used                                     : NO")

print("\nNEXT")
print("  Resume at Stage26-4A — exact Monday PCAP restoration only.")


STAGE26 RESET RECOVERY :: RUNTIME
Python executable : /usr/local/bin/python
Working directory : /kaggle/working
Kaggle input exists: True
GPU visibility env : 
GPU hidden for recovery cell: YES

STAGE26 RESET RECOVERY :: REPOSITORY RESTORATION
Existing repository found; auditing instead of recloning.

STAGE26 RESET RECOVERY :: SCIENTIFIC GIT ANCHOR
Expected HEAD : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Local HEAD    : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
origin/main   : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Commit subject: stage26: anchor CPU memory profiling
Repo clean    : True

[PASS] Exact Stage26-3B Git anchor restored.

STAGE26 RESET RECOVERY :: DURABLE ARTIFACT HASH GATE
Stage26 measurement protocol       PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
Stage26-3B memory receipt          PASS db4b39af6b78299b67a13fad1f218a701894ef0af6d705552146173ee22939ac
Stage26-3B package manifest        PASS ab73d0c3fab3905301acc28afbd01912d8021fe50e1e3b25ff818

In [3]:
# =============================================================================
# STAGE26-4A — EXACT MONDAY PCAP SOURCE RESTORATION
#
# CURRENT DURABLE PARENT:
#   347d93f21d454cc5bda2c45889c890c67cdf0ecc
#
# PURPOSE:
#   Restore the exact raw Monday PCAP required by the already-frozen
#   Stage26 extraction gate.
#
# ORDER:
#   1. Search /kaggle/input and known local locations.
#   2. Accept ONLY the exact frozen size + SHA256.
#   3. If absent, reacquire ONLY the frozen Hugging Face revision.
#   4. Create a Stage26 canonical symlink.
#   5. Write a local provenance receipt.
#
# NO:
#   - packet iteration
#   - flow reconstruction
#   - extraction timing
#   - model loading
#   - inference
#   - labels
#   - Thursday/Friday
#   - GPU
#   - Git modification
# =============================================================================

from __future__ import annotations

import os
import json
import hashlib
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. FROZEN IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

SOURCE_ROOT = (
    STAGE26_ROOT
    / "sources"
)

EXTRACTION_ROOT = (
    STAGE26_ROOT
    / "extraction"
)

SOURCE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


EXPECTED_HEAD = (
    "347d93f21d454cc5bda2c45889c890c67cdf0ecc"
)

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

EXPECTED_STAGE26_3B_RECEIPT_SHA256 = (
    "db4b39af6b78299b67a13fad1f218a701894ef0af6d705552146173ee22939ac"
)

EXPECTED_STAGE26_3B_MANIFEST_SHA256 = (
    "ab73d0c3fab3905301acc28afbd01912d8021fe50e1e3b25ff818244eeea0b06"
)


# =============================================================================
# 1. EXACT HISTORICAL MONDAY SOURCE
# =============================================================================

HF_REPO_ID = (
    "bvsam/cic-ids-2017"
)

HF_REPO_TYPE = (
    "dataset"
)

HF_REVISION = (
    "e810c1cc98270ec271a1df917b9de0786c33f343"
)

HF_FILENAME = (
    "pcap/Monday-WorkingHours.pcap"
)

SOURCE_BASENAME = (
    "Monday-WorkingHours.pcap"
)

EXPECTED_SOURCE_SIZE_BYTES = (
    10_822_507_416
)

EXPECTED_SOURCE_SHA256 = (
    "f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972"
)

HISTORICAL_RAW_PACKET_COUNT = (
    11_709_971
)

HISTORICAL_VALID_IPV4_PACKET_COUNT = (
    11_626_492
)

HISTORICAL_EXPORTABLE_FLOW_COUNT = (
    529_601
)


# =============================================================================
# 2. DURABLE PROVENANCE ARTIFACTS
# =============================================================================

PROTOCOL = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_0_protocol_lock/"
      "measurement_protocol.json"
)

STAGE26_3B_RECEIPT = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_3b_cpu_memory_profile/"
      "stage26_3b_memory_receipt.json"
)

STAGE26_3B_MANIFEST = (
    REPO
    / "results/stage26_deployment_profiling/"
      "stage26_3b_cpu_memory_profile/"
      "stage26_3b_memory_package_manifest.json"
)

GEOMETRY_PROFILE = (
    REPO
    / "results/stage20_1d_representation/"
      "stage20_1d2m_monday_exact_geometry_profile.json"
)

EXPECTED_GEOMETRY_PROFILE_SHA256 = (
    "3a26d6499334c12ea4e9272aef4250761c6cf4399e7fb4d33ef236f12d0b7272"
)


CANONICAL_SOURCE = (
    SOURCE_ROOT
    / SOURCE_BASENAME
)

SOURCE_RECEIPT = (
    EXTRACTION_ROOT
    / "stage26_4a_monday_source_restoration_receipt.json"
)


# =============================================================================
# 3. HELPERS
# =============================================================================

def banner(text):
    print("\n" + "=" * 116)
    print(text)
    print("=" * 116)


def run(cmd, *, cwd=None):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            "$ "
            + " ".join(map(str, cmd))
            + "\n\n"
            + p.stdout
        )

    return p.stdout.strip()


def git(*args):
    return run(
        ["git", *args],
        cwd=REPO,
    )


def sha256_file(path, *, progress=False):

    path = Path(path)

    h = hashlib.sha256()
    total = 0
    report_step = 2 * 1024**3
    next_report = report_step

    with path.open("rb") as f:

        while True:

            chunk = f.read(
                16 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

            total += len(chunk)

            if progress and total >= next_report:

                print(
                    f"  hashed {total / 1024**3:.2f} GiB",
                    flush=True,
                )

                next_report += report_step

    return h.hexdigest()


def atomic_json(path, obj):

    path = Path(path)
    tmp = Path(str(path) + ".tmp")

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write("\n")
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)


# =============================================================================
# 4. REPOSITORY / SCIENTIFIC ANCHOR GATE
# =============================================================================

banner(
    "STAGE26-4A :: REPOSITORY / SCIENTIFIC ANCHOR"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print("Expected HEAD :", EXPECTED_HEAD)
print("Local HEAD    :", head)
print("origin/main   :", remote)
print("Repository clean:", status == "")


if head != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected local HEAD."
    )

if remote != EXPECTED_HEAD:
    raise RuntimeError(
        "origin/main changed unexpectedly."
    )

if status:
    raise RuntimeError(
        "Repository must remain clean."
    )


# =============================================================================
# 5. DURABLE HASH GATE
# =============================================================================

banner(
    "STAGE26-4A :: DURABLE HASH GATE"
)


checks = [
    (
        "Stage26 protocol",
        PROTOCOL,
        EXPECTED_PROTOCOL_SHA256,
    ),
    (
        "Stage26-3B receipt",
        STAGE26_3B_RECEIPT,
        EXPECTED_STAGE26_3B_RECEIPT_SHA256,
    ),
    (
        "Stage26-3B manifest",
        STAGE26_3B_MANIFEST,
        EXPECTED_STAGE26_3B_MANIFEST_SHA256,
    ),
    (
        "Monday geometry profile",
        GEOMETRY_PROFILE,
        EXPECTED_GEOMETRY_PROFILE_SHA256,
    ),
]


for name, path, expected in checks:

    if not path.exists():
        raise FileNotFoundError(path)

    actual = sha256_file(path)

    ok = (
        actual == expected
    )

    print(
        f"{name:30s} "
        f"{'PASS' if ok else 'FAIL'} "
        f"{actual}"
    )

    if not ok:
        raise RuntimeError(
            f"Frozen artifact mismatch: {name}"
        )


# =============================================================================
# 6. HISTORICAL MACHINE-READABLE PROVENANCE GATE
# =============================================================================

banner(
    "STAGE26-4A :: HISTORICAL MONDAY IDENTITY"
)


geometry = json.loads(
    GEOMETRY_PROFILE.read_text(
        encoding="utf-8"
    )
)

provenance = geometry.get(
    "source_provenance"
)

if not isinstance(provenance, dict):
    raise RuntimeError(
        "Monday geometry profile lacks source_provenance."
    )


expected_provenance = {
    "source_filename":
        SOURCE_BASENAME,

    "expected_size_bytes":
        EXPECTED_SOURCE_SIZE_BYTES,

    "expected_sha256":
        EXPECTED_SOURCE_SHA256,

    "source_size_bytes":
        EXPECTED_SOURCE_SIZE_BYTES,

    "source_sha256":
        EXPECTED_SOURCE_SHA256,

    "source_revision_or_frozen_identity":
        HF_REVISION,

    "source_verified":
        True,
}


for key, expected in expected_provenance.items():

    actual = provenance.get(key)

    print(
        f"{key:42s}: {actual!r}"
    )

    if actual != expected:

        raise RuntimeError(
            f"Historical source mismatch: {key}"
        )


print(
    "\n[PASS] Historical Monday byte identity recovered."
)


# =============================================================================
# 7. SEARCH EXISTING KAGGLE INPUTS FIRST
# =============================================================================

banner(
    "STAGE26-4A :: SEARCH EXISTING LOCAL / ATTACHED SOURCES"
)


candidate_paths = []


search_roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]


for root in search_roots:

    if not root.exists():
        continue

    try:
        for path in root.rglob(
            SOURCE_BASENAME
        ):

            try:
                resolved = path.resolve(
                    strict=True
                )

            except Exception:
                continue

            if resolved.is_file():
                candidate_paths.append(
                    resolved
                )

    except Exception:
        pass


# De-duplicate resolved paths.
unique_candidates = []
seen = set()


for path in candidate_paths:

    key = str(path)

    if key not in seen:
        seen.add(key)
        unique_candidates.append(path)


print(
    "Candidates found:",
    len(unique_candidates),
)


for path in unique_candidates:

    print(
        " ",
        path,
        "bytes=",
        path.stat().st_size,
    )


# =============================================================================
# 8. VERIFY LOCAL CANDIDATES
# =============================================================================

selected_source = None
selected_sha = None
source_route = None


for path in unique_candidates:

    size = int(
        path.stat().st_size
    )

    if size != EXPECTED_SOURCE_SIZE_BYTES:

        print(
            "[SKIP wrong size]",
            path,
        )

        continue


    print(
        "\nExact-size candidate found:"
    )

    print(
        " ",
        path
    )

    print(
        "Computing SHA256..."
    )


    actual_sha = sha256_file(
        path,
        progress=True,
    )


    print(
        "SHA256:"
    )

    print(
        " ",
        actual_sha
    )


    if actual_sha == EXPECTED_SOURCE_SHA256:

        selected_source = path
        selected_sha = actual_sha

        source_route = (
            "EXISTING_KAGGLE_OR_LOCAL_BYTE_IDENTICAL_SOURCE"
        )

        print(
            "\n[PASS] Exact Monday PCAP already available."
        )

        break


    print(
        "[REJECT] SHA256 mismatch."
    )


# =============================================================================
# 9. REACQUIRE EXACT FROZEN HF REVISION ONLY IF ABSENT
# =============================================================================

if selected_source is None:

    banner(
        "STAGE26-4A :: FROZEN HUGGING FACE REACQUISITION"
    )


    print(
        "No byte-identical Monday PCAP found locally."
    )

    print(
        "Dataset :",
        HF_REPO_ID,
    )

    print(
        "Revision:",
        HF_REVISION,
    )

    print(
        "File    :",
        HF_FILENAME,
    )


    free_bytes = shutil.disk_usage(
        "/kaggle/working"
    ).free


    required_bytes = (
        EXPECTED_SOURCE_SIZE_BYTES
        +
        1024**3
    )


    print(
        "\nFree /kaggle/working:",
        f"{free_bytes / 1024**3:.2f} GiB",
    )

    print(
        "Required including 1 GiB headroom:",
        f"{required_bytes / 1024**3:.2f} GiB",
    )


    if free_bytes < required_bytes:

        raise RuntimeError(
            "Insufficient workspace free space for exact PCAP restoration."
        )


    try:

        from huggingface_hub import hf_hub_download

    except Exception as exc:

        raise RuntimeError(
            "huggingface_hub is unavailable. "
            "Do not substitute another source."
        ) from exc


    HF_CACHE = Path(
        "/kaggle/working/stage26_hf_cache"
    )

    HF_CACHE.mkdir(
        parents=True,
        exist_ok=True,
    )


    downloaded_path = Path(
        hf_hub_download(
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
            filename=HF_FILENAME,
            revision=HF_REVISION,
            cache_dir=str(HF_CACHE),
        )
    ).resolve(
        strict=True
    )


    downloaded_size = int(
        downloaded_path.stat().st_size
    )


    print(
        "\nResolved HF source:"
    )

    print(
        " ",
        downloaded_path
    )

    print(
        "Bytes:",
        downloaded_size,
    )


    if downloaded_size != EXPECTED_SOURCE_SIZE_BYTES:

        raise RuntimeError(
            "Reacquired Monday PCAP size mismatch."
        )


    print(
        "\nComputing complete PCAP SHA256..."
    )


    downloaded_sha = sha256_file(
        downloaded_path,
        progress=True,
    )


    print(
        "SHA256:"
    )

    print(
        " ",
        downloaded_sha
    )


    if downloaded_sha != EXPECTED_SOURCE_SHA256:

        raise RuntimeError(
            "Reacquired Monday PCAP SHA256 mismatch."
        )


    selected_source = downloaded_path
    selected_sha = downloaded_sha

    source_route = (
        "HUGGINGFACE_EXACT_FROZEN_REVISION"
    )


    print(
        "\n[PASS] Exact frozen Monday PCAP reacquired."
    )


# =============================================================================
# 10. CANONICAL STAGE26 POINTER
# =============================================================================

banner(
    "STAGE26-4A :: CANONICAL SOURCE POINTER"
)


if (
    CANONICAL_SOURCE.exists()
    or
    CANONICAL_SOURCE.is_symlink()
):

    CANONICAL_SOURCE.unlink()


CANONICAL_SOURCE.symlink_to(
    selected_source
)


resolved = CANONICAL_SOURCE.resolve(
    strict=True
)


if resolved != selected_source:

    raise RuntimeError(
        "Canonical source does not resolve to verified PCAP."
    )


if int(
    resolved.stat().st_size
) != EXPECTED_SOURCE_SIZE_BYTES:

    raise RuntimeError(
        "Canonical source size mismatch."
    )


print(
    "Canonical Stage26 source:"
)

print(
    " ",
    CANONICAL_SOURCE
)

print(
    "Resolves to:"
)

print(
    " ",
    resolved
)


# =============================================================================
# 11. LOCAL SOURCE RESTORATION RECEIPT
# =============================================================================

receipt = {
    "schema":
        "stage26_4a_source_restoration_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-4A",

    "status":
        "PASS_EXACT_SOURCE_RESTORED",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_HEAD,

    "source": {
        "dataset_repo_id":
            HF_REPO_ID,

        "dataset_revision":
            HF_REVISION,

        "filename":
            HF_FILENAME,

        "size_bytes":
            EXPECTED_SOURCE_SIZE_BYTES,

        "sha256":
            EXPECTED_SOURCE_SHA256,

        "historical_raw_packet_count":
            HISTORICAL_RAW_PACKET_COUNT,

        "historical_valid_ipv4_packet_count":
            HISTORICAL_VALID_IPV4_PACKET_COUNT,

        "historical_exportable_flow_count":
            HISTORICAL_EXPORTABLE_FLOW_COUNT,
    },

    "restoration": {
        "route":
            source_route,

        "verified_source_path":
            str(selected_source),

        "canonical_stage26_path":
            str(CANONICAL_SOURCE),

        "resolved_path":
            str(resolved),

        "verified_size_bytes":
            int(
                resolved.stat().st_size
            ),

        "verified_sha256":
            selected_sha,

        "full_sha256_verification":
            True,
    },

    "scientific_boundary": {
        "packet_iteration":
            False,

        "packet_parsing":
            False,

        "flow_reconstruction":
            False,

        "extraction_timing":
            False,

        "throughput_computed":
            False,

        "labels_read":
            False,

        "Thursday_accessed":
            False,

        "Friday_accessed":
            False,

        "models_loaded":
            False,

        "inference":
            False,

        "gpu_used":
            False,

        "git_modified":
            False,
    },

    "next_action":
        (
            "STAGE26-4B_FREEZE_EXTRACTION_SUBPROTOCOL_"
            "BEFORE_FIRST_EXTRACTION_TIMING"
        ),
}


atomic_json(
    SOURCE_RECEIPT,
    receipt,
)


receipt_sha = sha256_file(
    SOURCE_RECEIPT
)


# =============================================================================
# 12. FINAL AUDIT
# =============================================================================

banner(
    "STAGE26-4A :: FINAL AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "Local HEAD :",
    final_head,
)

print(
    "Remote HEAD:",
    final_remote,
)

print(
    "Repo clean :",
    final_status == "",
)


if final_head != EXPECTED_HEAD:

    raise RuntimeError(
        "Git HEAD changed unexpectedly."
    )


if final_remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed unexpectedly."
    )


if final_status:

    raise RuntimeError(
        "Repository changed during Stage26-4A."
    )


banner(
    "STAGE26-4A COMPLETE"
)


print(
    "EXACT SOURCE:"
)

print(
    "  ",
    SOURCE_BASENAME
)

print(
    "  bytes :",
    EXPECTED_SOURCE_SIZE_BYTES
)

print(
    "  SHA256:",
    EXPECTED_SOURCE_SHA256
)

print(
    "  packets:",
    HISTORICAL_RAW_PACKET_COUNT
)


print(
    "\nRESTORATION ROUTE:"
)

print(
    " ",
    source_route
)


print(
    "\nCANONICAL PATH:"
)

print(
    " ",
    CANONICAL_SOURCE
)


print(
    "\nSOURCE RECEIPT:"
)

print(
    " ",
    SOURCE_RECEIPT
)

print(
    " SHA256:",
    receipt_sha
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  EXACT MONDAY SOURCE RESTORED : PASS"
)

print(
    "  BYTE IDENTITY VERIFIED       : PASS"
)

print(
    "  PACKETS PARSED               : NO"
)

print(
    "  EXTRACTION TIMED             : NO"
)

print(
    "  MODEL INFERENCE              : NO"
)

print(
    "  LABELS READ                  : NO"
)

print(
    "  THURSDAY/FRIDAY ACCESSED     : NO"
)

print(
    "  GPU USED                     : NO"
)

print(
    "  GIT MODIFIED                 : NO"
)


print(
    "\nNEXT:"
)

print(
    "  STAGE26-4B — freeze deterministic PCAP bounds/sample,"
)

print(
    "  I/O + serialization + startup semantics and extractor versions"
)

print(
    "  BEFORE the first extraction timing."
)


STAGE26-4A :: REPOSITORY / SCIENTIFIC ANCHOR
Expected HEAD : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Local HEAD    : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
origin/main   : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Repository clean: True

STAGE26-4A :: DURABLE HASH GATE
Stage26 protocol               PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
Stage26-3B receipt             PASS db4b39af6b78299b67a13fad1f218a701894ef0af6d705552146173ee22939ac
Stage26-3B manifest            PASS ab73d0c3fab3905301acc28afbd01912d8021fe50e1e3b25ff818244eeea0b06
Monday geometry profile        PASS 3a26d6499334c12ea4e9272aef4250761c6cf4399e7fb4d33ef236f12d0b7272

STAGE26-4A :: HISTORICAL MONDAY IDENTITY
source_filename                           : 'Monday-WorkingHours.pcap'
expected_size_bytes                       : 10822507416
expected_sha256                           : 'f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972'
source_size_bytes                         :

pcap/Monday-WorkingHours.pcap:   0%|          | 0.00/10.8G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [4]:
# =============================================================================
# STAGE26-4A-RECOVERY-A
# DIAGNOSE STALLED MONDAY PCAP DOWNLOAD
#
# NO NETWORK
# NO DELETION
# NO PCAP PARSING
# NO EXTRACTION
# NO TIMING
# NO MODEL WORK
#
# Purpose:
#   Inspect the partially downloaded Hugging Face/Xet cache after the
#   Stage26-4A transfer stalled at ~83%.
# =============================================================================

from __future__ import annotations

import os
import sys
import shutil
import subprocess
from pathlib import Path


CACHE_ROOT = Path(
    "/kaggle/working/stage26_hf_cache"
)

EXPECTED_SIZE = 10_822_507_416

EXPECTED_SHA = (
    "f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972"
)


def banner(text):
    print("\n" + "=" * 110)
    print(text)
    print("=" * 110)


def human(n):
    if n is None:
        return "N/A"

    for unit in ["B", "KiB", "MiB", "GiB", "TiB"]:
        if abs(n) < 1024 or unit == "TiB":
            return f"{n:.3f} {unit}"
        n /= 1024


# =============================================================================
# 1. DISK STATE
# =============================================================================

banner("STAGE26-4A-RECOVERY-A :: DISK STATE")

usage = shutil.disk_usage(
    "/kaggle/working"
)

print(
    "Total:",
    human(usage.total)
)

print(
    "Used :",
    human(usage.used)
)

print(
    "Free :",
    human(usage.free)
)

print(
    "Expected Monday PCAP:",
    human(EXPECTED_SIZE)
)


# =============================================================================
# 2. HUGGING FACE PACKAGE STATE
# =============================================================================

banner("STAGE26-4A-RECOVERY-A :: HF SOFTWARE STATE")

try:
    import huggingface_hub

    print(
        "huggingface_hub:",
        huggingface_hub.__version__,
    )

except Exception as exc:
    print(
        "huggingface_hub import failed:",
        repr(exc),
    )


try:
    import hf_xet

    print(
        "hf_xet:",
        getattr(
            hf_xet,
            "__version__",
            "<installed/version unavailable>",
        ),
    )

    print(
        "hf_xet installed: YES"
    )

except Exception as exc:

    print(
        "hf_xet installed: NO"
    )

    print(
        "detail:",
        repr(exc),
    )


print(
    "HF_HUB_DISABLE_XET:",
    os.environ.get(
        "HF_HUB_DISABLE_XET",
        "<unset>",
    ),
)

print(
    "HF_XET_HIGH_PERFORMANCE:",
    os.environ.get(
        "HF_XET_HIGH_PERFORMANCE",
        "<unset>",
    ),
)

print(
    "HF_HUB_DOWNLOAD_TIMEOUT:",
    os.environ.get(
        "HF_HUB_DOWNLOAD_TIMEOUT",
        "<unset>",
    ),
)


# =============================================================================
# 3. CACHE EXISTENCE / TOTAL SIZE
# =============================================================================

banner("STAGE26-4A-RECOVERY-A :: CACHE STATE")

print(
    "Cache root:",
    CACHE_ROOT
)

print(
    "Exists:",
    CACHE_ROOT.exists()
)


if CACHE_ROOT.exists():

    total_cache_bytes = 0
    file_count = 0

    for path in CACHE_ROOT.rglob("*"):

        try:

            if path.is_file():

                total_cache_bytes += (
                    path.stat().st_size
                )

                file_count += 1

        except Exception:
            pass


    print(
        "Cache files:",
        file_count
    )

    print(
        "Cache apparent bytes:",
        human(total_cache_bytes)
    )


# =============================================================================
# 4. FIND LARGE / PARTIAL OBJECTS
# =============================================================================

banner("STAGE26-4A-RECOVERY-A :: LARGE CACHE OBJECTS")

records = []


if CACHE_ROOT.exists():

    for path in CACHE_ROOT.rglob("*"):

        try:

            if not path.is_file():
                continue

            size = int(
                path.stat().st_size
            )

            # Show files >= 50 MiB plus anything obviously incomplete.
            if (
                size >= 50 * 1024**2
                or
                ".incomplete" in path.name
                or
                ".partial" in path.name
                or
                "xet" in str(path).lower()
            ):

                records.append(
                    (
                        size,
                        path,
                    )
                )

        except Exception:
            pass


records.sort(
    key=lambda x: x[0],
    reverse=True,
)


if not records:

    print(
        "No >=50 MiB / incomplete cache files found."
    )

else:

    for size, path in records[:50]:

        print(
            f"{human(size):>12s}  {path}"
        )


# =============================================================================
# 5. SEARCH FOR EXACT OR NEAR-EXACT MONDAY OBJECT
# =============================================================================

banner("STAGE26-4A-RECOVERY-A :: MONDAY OBJECT SEARCH")

possible = []


search_roots = [
    Path("/kaggle/working/stage26_hf_cache"),
    Path("/kaggle/working"),
]


seen = set()


for root in search_roots:

    if not root.exists():
        continue

    for path in root.rglob("*"):

        try:

            if not path.is_file():
                continue

            resolved = path.resolve()

            key = str(resolved)

            if key in seen:
                continue

            seen.add(key)

            size = int(
                resolved.stat().st_size
            )

            # Anything at least 1 GiB is relevant to diagnosing this transfer.
            if size >= 1024**3:

                possible.append(
                    (
                        size,
                        resolved,
                    )
                )

        except Exception:
            pass


possible.sort(
    key=lambda x: x[0],
    reverse=True,
)


for size, path in possible[:30]:

    fraction = (
        size
        /
        EXPECTED_SIZE
    )

    print(
        f"{human(size):>12s} "
        f"({fraction:7.2%} of expected)  "
        f"{path}"
    )


if not possible:

    print(
        "No >=1 GiB files found."
    )


# =============================================================================
# 6. CHECK WHETHER HF ALREADY CONSIDERS FILE COMPLETE — OFFLINE ONLY
# =============================================================================

banner("STAGE26-4A-RECOVERY-A :: OFFLINE CACHE RESOLUTION")

try:

    from huggingface_hub import hf_hub_download

    cached = hf_hub_download(
        repo_id="bvsam/cic-ids-2017",
        repo_type="dataset",
        filename="pcap/Monday-WorkingHours.pcap",
        revision="e810c1cc98270ec271a1df917b9de0786c33f343",
        cache_dir=str(
            CACHE_ROOT
        ),
        local_files_only=True,
    )

    cached_path = Path(
        cached
    ).resolve(
        strict=True
    )

    print(
        "HF reports COMPLETE cached object:"
    )

    print(
        " ",
        cached_path
    )

    print(
        "Size:",
        human(
            cached_path.stat().st_size
        )
    )


except Exception as exc:

    print(
        "HF does NOT currently consider the file complete."
    )

    print(
        "Exception:",
        type(exc).__name__,
        str(exc)[:1000],
    )


# =============================================================================
# 7. PROCESS CHECK
# =============================================================================

banner("STAGE26-4A-RECOVERY-A :: DOWNLOAD PROCESS CHECK")

try:

    ps = subprocess.run(
        [
            "ps",
            "-eo",
            "pid,ppid,stat,etime,cmd",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    ).stdout


    matches = [
        line
        for line in ps.splitlines()
        if (
            "huggingface" in line.lower()
            or
            "hf_xet" in line.lower()
            or
            "xet" in line.lower()
        )
    ]


    if matches:

        print(
            "\n".join(matches)
        )

    else:

        print(
            "No Hugging Face/Xet transfer process found."
        )


except Exception as exc:

    print(
        "Process check failed:",
        repr(exc),
    )


# =============================================================================
# 8. CLOSURE
# =============================================================================

banner("STAGE26-4A-RECOVERY-A COMPLETE")

print(
    "NO FILES DELETED."
)

print(
    "NO NETWORK REQUESTS MADE."
)

print(
    "NO PCAP PARSING PERFORMED."
)

print(
    "NO EXTRACTION MEASURED."
)

print(
    "\nPaste the COMPLETE output of this diagnostic cell."
)


STAGE26-4A-RECOVERY-A :: DISK STATE
Total: 19.518 GiB
Used : 10.595 GiB
Free : 8.908 GiB
Expected Monday PCAP: 10.079 GiB

STAGE26-4A-RECOVERY-A :: HF SOFTWARE STATE
huggingface_hub: 1.11.0
hf_xet: <installed/version unavailable>
hf_xet installed: YES
HF_HUB_DISABLE_XET: <unset>
HF_XET_HIGH_PERFORMANCE: <unset>
HF_HUB_DOWNLOAD_TIMEOUT: <unset>

STAGE26-4A-RECOVERY-A :: CACHE STATE
Cache root: /kaggle/working/stage26_hf_cache
Exists: True
Cache files: 2
Cache apparent bytes: 8.403 GiB

STAGE26-4A-RECOVERY-A :: LARGE CACHE OBJECTS
   8.403 GiB  /kaggle/working/stage26_hf_cache/datasets--bvsam--cic-ids-2017/blobs/f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972.incomplete

STAGE26-4A-RECOVERY-A :: MONDAY OBJECT SEARCH
   8.403 GiB ( 83.37% of expected)  /kaggle/working/stage26_hf_cache/datasets--bvsam--cic-ids-2017/blobs/f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972.incomplete

STAGE26-4A-RECOVERY-A :: OFFLINE CACHE RESOLUTION
HF does NOT currently cons

In [5]:
# =============================================================================
# STAGE26-4A-RECOVERY-B
# RESUME EXACT MONDAY PCAP DOWNLOAD WITH XET DISABLED
#
# EXISTING STATE:
#   partial .incomplete blob ≈ 8.403 GiB / 83.37%
#
# STRATEGY:
#   - DO NOT delete partial cache.
#   - Spawn a fresh Python process.
#   - Set HF_HUB_DISABLE_XET=1 BEFORE huggingface_hub import.
#   - Set HF_HUB_DOWNLOAD_TIMEOUT=300.
#   - Reuse exact same Hugging Face cache.
#   - Verify final size + SHA256.
#
# NO:
#   - packet parsing
#   - flow reconstruction
#   - extraction timing
#   - model inference
#   - labels
#   - GPU
#   - Git modification
# =============================================================================

from __future__ import annotations

import os
import sys
import json
import time
import hashlib
import shutil
import subprocess
import textwrap
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

SOURCE_ROOT = (
    STAGE26_ROOT
    / "sources"
)

EXTRACTION_ROOT = (
    STAGE26_ROOT
    / "extraction"
)

SOURCE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


EXPECTED_HEAD = (
    "347d93f21d454cc5bda2c45889c890c67cdf0ecc"
)

HF_CACHE = Path(
    "/kaggle/working/stage26_hf_cache"
)

HF_REPO_ID = (
    "bvsam/cic-ids-2017"
)

HF_REVISION = (
    "e810c1cc98270ec271a1df917b9de0786c33f343"
)

HF_FILENAME = (
    "pcap/Monday-WorkingHours.pcap"
)

SOURCE_BASENAME = (
    "Monday-WorkingHours.pcap"
)

EXPECTED_SIZE = (
    10_822_507_416
)

EXPECTED_SHA = (
    "f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972"
)

EXPECTED_PACKET_COUNT = (
    11_709_971
)

EXPECTED_VALID_IPV4_COUNT = (
    11_626_492
)

EXPECTED_EXPORTABLE_FLOW_COUNT = (
    529_601
)


PARTIAL_PATH = (
    HF_CACHE
    / "datasets--bvsam--cic-ids-2017"
    / "blobs"
    / (
        EXPECTED_SHA
        + ".incomplete"
    )
)

CANONICAL_SOURCE = (
    SOURCE_ROOT
    / SOURCE_BASENAME
)

SOURCE_RECEIPT = (
    EXTRACTION_ROOT
    / "stage26_4a_monday_source_restoration_receipt.json"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 112
    )

    print(text)

    print(
        "=" * 112
    )


def human(n):

    n = float(n)

    for unit in [
        "B",
        "KiB",
        "MiB",
        "GiB",
        "TiB",
    ]:

        if n < 1024 or unit == "TiB":

            return f"{n:.3f} {unit}"

        n /= 1024


def sha256_file(
    path,
    *,
    progress=False,
):

    path = Path(path)

    h = hashlib.sha256()

    total = 0

    next_report = (
        2 * 1024**3
    )


    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                16 * 1024 * 1024
            )

            if not chunk:
                break


            h.update(
                chunk
            )

            total += len(
                chunk
            )


            if (
                progress
                and
                total >= next_report
            ):

                print(
                    f"  hashed "
                    f"{total / 1024**3:.2f} GiB",
                    flush=True,
                )

                next_report += (
                    2 * 1024**3
                )


    return h.hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(path)

    tmp = Path(
        str(path)
        + ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )


    os.replace(
        tmp,
        path,
    )


def git(*args):

    p = subprocess.run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


# =============================================================================
# 2. SCIENTIFIC GIT ANCHOR
# =============================================================================

banner(
    "STAGE26-4A-RECOVERY-B :: GIT ANCHOR"
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD:",
    EXPECTED_HEAD
)

print(
    "Local HEAD   :",
    head
)

print(
    "origin/main  :",
    remote
)

print(
    "Repo clean   :",
    status == ""
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected local HEAD."
    )


if remote != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected origin/main."
    )


if status:

    raise RuntimeError(
        "Repository must remain clean."
    )


# =============================================================================
# 3. PARTIAL CACHE AUDIT
# =============================================================================

banner(
    "STAGE26-4A-RECOVERY-B :: PARTIAL CACHE AUDIT"
)


print(
    "Partial path:"
)

print(
    " ",
    PARTIAL_PATH
)


if PARTIAL_PATH.exists():

    partial_size_before = int(
        PARTIAL_PATH.stat().st_size
    )


    print(
        "Partial exists: YES"
    )

    print(
        "Partial size  :",
        human(
            partial_size_before
        )
    )

    print(
        "Completion    :",
        f"{partial_size_before / EXPECTED_SIZE:.2%}"
    )


else:

    partial_size_before = 0


    print(
        "Partial exists: NO"
    )

    print(
        "The retry will still use the exact frozen source."
    )


disk_before = shutil.disk_usage(
    "/kaggle/working"
)


print(
    "\nWorkspace free:",
    human(
        disk_before.free
    )
)


remaining_bytes = max(
    0,
    EXPECTED_SIZE
    -
    partial_size_before,
)


print(
    "Remaining source bytes:",
    human(
        remaining_bytes
    )
)


# =============================================================================
# 4. CREATE FRESH-SUBPROCESS DOWNLOADER
# =============================================================================

banner(
    "STAGE26-4A-RECOVERY-B :: HTTP FALLBACK / RESUME"
)


downloader_script = (
    STAGE26_ROOT
    / "runtime"
    / "stage26_4a_http_resume.py"
)

downloader_script.parent.mkdir(
    parents=True,
    exist_ok=True,
)


script = r'''
import os

# MUST be set before importing huggingface_hub.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"

from pathlib import Path
from huggingface_hub import hf_hub_download


CACHE = "/kaggle/working/stage26_hf_cache"

print(
    "Fresh subprocess environment:",
    flush=True,
)

print(
    "  HF_HUB_DISABLE_XET      =",
    os.environ.get("HF_HUB_DISABLE_XET"),
    flush=True,
)

print(
    "  HF_HUB_DOWNLOAD_TIMEOUT =",
    os.environ.get("HF_HUB_DOWNLOAD_TIMEOUT"),
    flush=True,
)

print(
    "  HF_HUB_ETAG_TIMEOUT     =",
    os.environ.get("HF_HUB_ETAG_TIMEOUT"),
    flush=True,
)

print(
    "\nCalling hf_hub_download using existing cache...",
    flush=True,
)


path = hf_hub_download(
    repo_id="bvsam/cic-ids-2017",
    repo_type="dataset",
    filename="pcap/Monday-WorkingHours.pcap",
    revision="e810c1cc98270ec271a1df917b9de0786c33f343",
    cache_dir=CACHE,
    local_files_only=False,
    force_download=False,
)


resolved = Path(
    path
).resolve(
    strict=True
)


print(
    "\nDOWNLOAD_RETURNED",
    flush=True,
)

print(
    str(resolved),
    flush=True,
)

print(
    "SIZE_BYTES",
    resolved.stat().st_size,
    flush=True,
)
'''


downloader_script.write_text(
    textwrap.dedent(
        script
    ).lstrip(),
    encoding="utf-8",
)


# =============================================================================
# 5. RUN FRESH PROCESS WITH STREAMED OUTPUT
# =============================================================================

env = os.environ.copy()

# Also set at process level for defense in depth.
env[
    "HF_HUB_DISABLE_XET"
] = "1"

env[
    "HF_HUB_DOWNLOAD_TIMEOUT"
] = "300"

env[
    "HF_HUB_ETAG_TIMEOUT"
] = "60"


print(
    "Launching fresh Python subprocess..."
)

print(
    "Existing partial file is NOT deleted."
)

print()


proc = subprocess.Popen(
    [
        sys.executable,
        "-u",
        str(
            downloader_script
        ),
    ],
    cwd=REPO,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)


captured_lines = []


while True:

    line = proc.stdout.readline()

    if line:

        print(
            line,
            end="",
            flush=True,
        )

        captured_lines.append(
            line
        )


    if (
        proc.poll() is not None
        and
        not line
    ):

        break


returncode = proc.wait()


print(
    "\nSubprocess return code:",
    returncode
)


if returncode != 0:

    current_partial_size = (
        int(
            PARTIAL_PATH.stat().st_size
        )
        if PARTIAL_PATH.exists()
        else 0
    )


    print(
        "\nCurrent partial size:",
        human(
            current_partial_size
        )
    )


    print(
        "Current completion:",
        f"{current_partial_size / EXPECTED_SIZE:.2%}"
        if EXPECTED_SIZE
        else "N/A",
    )


    raise RuntimeError(
        "HTTP/Xet-disabled resume did not complete.\n"
        "DO NOT delete the partial cache; paste this full output."
    )


# =============================================================================
# 6. OFFLINE-RESOLVE COMPLETED CACHE ENTRY
# =============================================================================

banner(
    "STAGE26-4A-RECOVERY-B :: OFFLINE COMPLETION GATE"
)


# Use another fresh subprocess with network disabled semantically via
# local_files_only=True. This verifies Hugging Face now considers it complete.
offline_script = (
    STAGE26_ROOT
    / "runtime"
    / "stage26_4a_offline_resolve.py"
)


offline_script.write_text(
    r'''
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

from pathlib import Path
from huggingface_hub import hf_hub_download


path = hf_hub_download(
    repo_id="bvsam/cic-ids-2017",
    repo_type="dataset",
    filename="pcap/Monday-WorkingHours.pcap",
    revision="e810c1cc98270ec271a1df917b9de0786c33f343",
    cache_dir="/kaggle/working/stage26_hf_cache",
    local_files_only=True,
)


resolved = Path(path).resolve(strict=True)

print(str(resolved))
print(resolved.stat().st_size)
'''.lstrip(),
    encoding="utf-8",
)


offline = subprocess.run(
    [
        sys.executable,
        str(
            offline_script
        ),
    ],
    cwd=REPO,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)


print(
    offline.stdout
)


if offline.returncode != 0:

    raise RuntimeError(
        "Download process returned success but Hugging Face "
        "does not consider the cache entry complete."
    )


offline_lines = [
    x.strip()
    for x in offline.stdout.splitlines()
    if x.strip()
]


if len(
    offline_lines
) < 2:

    raise RuntimeError(
        "Unable to resolve completed cached source."
    )


completed_path = Path(
    offline_lines[
        -2
    ]
).resolve(
    strict=True
)


completed_size = int(
    offline_lines[
        -1
    ]
)


print(
    "Completed object:"
)

print(
    " ",
    completed_path
)

print(
    "Size:"
)

print(
    " ",
    human(
        completed_size
    )
)


if completed_size != EXPECTED_SIZE:

    raise RuntimeError(
        "Completed PCAP size mismatch."
    )


# =============================================================================
# 7. FULL SCIENTIFIC BYTE HASH
# =============================================================================

banner(
    "STAGE26-4A-RECOVERY-B :: FULL SHA256 GATE"
)


print(
    "Expected size:"
)

print(
    " ",
    EXPECTED_SIZE
)


print(
    "Expected SHA256:"
)

print(
    " ",
    EXPECTED_SHA
)


print(
    "\nHashing complete 10.08 GiB source..."
)


actual_sha = sha256_file(
    completed_path,
    progress=True,
)


print(
    "\nActual SHA256:"
)

print(
    " ",
    actual_sha
)


if actual_sha != EXPECTED_SHA:

    raise RuntimeError(
        "Completed source SHA256 mismatch."
    )


print(
    "\n[PASS] Exact historical Monday bytes restored."
)


# =============================================================================
# 8. CANONICAL STAGE26 SOURCE POINTER
# =============================================================================

banner(
    "STAGE26-4A-RECOVERY-B :: CANONICAL SOURCE"
)


if (
    CANONICAL_SOURCE.exists()
    or
    CANONICAL_SOURCE.is_symlink()
):

    CANONICAL_SOURCE.unlink()


CANONICAL_SOURCE.symlink_to(
    completed_path
)


resolved = CANONICAL_SOURCE.resolve(
    strict=True
)


if resolved != completed_path:

    raise RuntimeError(
        "Canonical source pointer mismatch."
    )


print(
    "Canonical source:"
)

print(
    " ",
    CANONICAL_SOURCE
)

print(
    "Resolves to:"
)

print(
    " ",
    resolved
)


# =============================================================================
# 9. STAGE26-4A RESTORATION RECEIPT
# =============================================================================

receipt = {
    "schema":
        "stage26_4a_monday_source_restoration_receipt_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-4A",

    "status":
        "PASS_EXACT_SOURCE_RESTORED",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_HEAD,

    "source_identity": {
        "repo_id":
            HF_REPO_ID,

        "revision":
            HF_REVISION,

        "filename":
            HF_FILENAME,

        "size_bytes":
            EXPECTED_SIZE,

        "sha256":
            EXPECTED_SHA,

        "historical_raw_packet_count":
            EXPECTED_PACKET_COUNT,

        "historical_valid_ipv4_packet_count":
            EXPECTED_VALID_IPV4_COUNT,

        "historical_exportable_flow_count":
            EXPECTED_EXPORTABLE_FLOW_COUNT,
    },

    "recovery": {
        "prior_xet_transfer_stalled":
            True,

        "partial_blob_preserved":
            True,

        "partial_size_before_retry":
            partial_size_before,

        "partial_fraction_before_retry":
            (
                partial_size_before
                /
                EXPECTED_SIZE
            ),

        "retry_method":
            "HF_HUB_DISABLE_XET_HTTP_FALLBACK_IN_FRESH_SUBPROCESS",

        "HF_HUB_DOWNLOAD_TIMEOUT":
            300,

        "HF_HUB_ETAG_TIMEOUT":
            60,

        "completed_cache_path":
            str(
                completed_path
            ),

        "canonical_stage26_path":
            str(
                CANONICAL_SOURCE
            ),

        "verified_size_bytes":
            completed_size,

        "verified_sha256":
            actual_sha,
    },

    "scientific_boundary": {
        "packet_iteration":
            False,

        "packet_parsing":
            False,

        "flow_reconstruction":
            False,

        "extraction_timing":
            False,

        "throughput_computed":
            False,

        "model_loaded":
            False,

        "inference":
            False,

        "labels_read":
            False,

        "Thursday_accessed":
            False,

        "Friday_accessed":
            False,

        "gpu_used":
            False,

        "git_modified":
            False,
    },

    "next_action":
        (
            "STAGE26-4B_FREEZE_EXTRACTION_SUBPROTOCOL_"
            "BEFORE_FIRST_EXTRACTION_TIMING"
        ),
}


atomic_json(
    SOURCE_RECEIPT,
    receipt,
)


receipt_sha = sha256_file(
    SOURCE_RECEIPT
)


# =============================================================================
# 10. FINAL AUDIT
# =============================================================================

banner(
    "STAGE26-4A-RECOVERY-B :: FINAL AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


disk_after = shutil.disk_usage(
    "/kaggle/working"
)


print(
    "Local HEAD :",
    final_head
)

print(
    "Remote HEAD:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


print(
    "\nWorkspace free:"
)

print(
    " ",
    human(
        disk_after.free
    )
)


if final_head != EXPECTED_HEAD:

    raise RuntimeError(
        "HEAD changed unexpectedly."
    )


if final_remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed unexpectedly."
    )


if final_status:

    raise RuntimeError(
        "Git repository changed unexpectedly."
    )


banner(
    "STAGE26-4A SOURCE RESTORATION COMPLETE"
)


print(
    "SOURCE:"
)

print(
    " ",
    SOURCE_BASENAME
)


print(
    "\nSIZE:"
)

print(
    " ",
    EXPECTED_SIZE
)


print(
    "\nSHA256:"
)

print(
    " ",
    actual_sha
)


print(
    "\nHISTORICAL PACKETS:"
)

print(
    " ",
    EXPECTED_PACKET_COUNT
)


print(
    "\nSOURCE RECEIPT:"
)

print(
    " ",
    SOURCE_RECEIPT
)

print(
    " SHA256:",
    receipt_sha
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  MONDAY PCAP RESTORED       : PASS"
)

print(
    "  BYTE IDENTITY              : PASS"
)

print(
    "  PACKETS PARSED             : NO"
)

print(
    "  EXTRACTION TIMING          : NO"
)

print(
    "  MODELS LOADED              : NO"
)

print(
    "  HOLDOUT ACCESSED           : NO"
)

print(
    "  GPU USED                   : NO"
)


print(
    "\nNEXT:"
)

print(
    "  STAGE26-4B — freeze the bounded PCAP extraction sample"
)

print(
    "  and all I/O/startup/serialization/extractor semantics"
)

print(
    "  BEFORE the first timed extraction measurement."
)


STAGE26-4A-RECOVERY-B :: GIT ANCHOR
Expected HEAD: 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Local HEAD   : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
origin/main  : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Repo clean   : True

STAGE26-4A-RECOVERY-B :: PARTIAL CACHE AUDIT
Partial path:
  /kaggle/working/stage26_hf_cache/datasets--bvsam--cic-ids-2017/blobs/f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972.incomplete
Partial exists: YES
Partial size  : 8.403 GiB
Completion    : 83.37%

Workspace free: 8.908 GiB
Remaining source bytes: 1.676 GiB

STAGE26-4A-RECOVERY-B :: HTTP FALLBACK / RESUME
Launching fresh Python subprocess...
Existing partial file is NOT deleted.

Fresh subprocess environment:
  HF_HUB_DISABLE_XET      = 1
  HF_HUB_DOWNLOAD_TIMEOUT = 300
  HF_HUB_ETAG_TIMEOUT     = 60

Calling hf_hub_download using existing cache...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:731: UserWarning: Not enough free disk space to download the file. T

In [6]:
# =============================================================================
# STAGE26-4B0 — EXTRACTION IMPLEMENTATION / RUNTIME INVENTORY
#
# NO:
#   - PCAP packet iteration
#   - extraction
#   - timing
#   - labels
#   - model loading
#   - inference
#   - GPU
#   - main repository modification
#
# PURPOSE:
#   Identify exactly what extractor/runtime can be frozen for Stage26-4B.
# =============================================================================

from __future__ import annotations

import os
import json
import hashlib
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. PATHS / IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

EXTRACTION_ROOT = (
    STAGE26_ROOT
    / "extraction"
)

EXTERNAL_ROOT = (
    STAGE26_ROOT
    / "external"
)

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

EXTERNAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


EXPECTED_HEAD = (
    "347d93f21d454cc5bda2c45889c890c67cdf0ecc"
)


MONDAY_PCAP = (
    STAGE26_ROOT
    / "sources"
    / "Monday-WorkingHours.pcap"
)

EXPECTED_PCAP_SIZE = (
    10_822_507_416
)

EXPECTED_PCAP_SHA256 = (
    "f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972"
)


SOURCE_RECEIPT = (
    EXTRACTION_ROOT
    / "stage26_4a_monday_source_restoration_receipt.json"
)

EXPECTED_SOURCE_RECEIPT_SHA256 = (
    "9a6fdb032e7be7e047a66cf08b5d59ee5523480eb20faf6975403fafe4a91b0a"
)


GEOMETRY_PROFILE = (
    REPO
    / "results"
    / "stage20_1d_representation"
    / "stage20_1d2m_monday_exact_geometry_profile.json"
)

EXPECTED_GEOMETRY_SHA256 = (
    "3a26d6499334c12ea4e9272aef4250761c6cf4399e7fb4d33ef236f12d0b7272"
)


CICFLOWMETER_REMOTE = (
    "https://github.com/ahlashkari/CICFlowMeter.git"
)

CICFLOWMETER_COMMIT = (
    "eaa853dd82f08ba5288bb7f295b471de7313f883"
)

CICFLOWMETER_DIR = (
    EXTERNAL_ROOT
    / "CICFlowMeter"
)


INVENTORY_RECEIPT = (
    EXTRACTION_ROOT
    / "stage26_4b0_extractor_runtime_inventory.json"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):
    print("\n" + "=" * 116)
    print(text)
    print("=" * 116)


def run(cmd, *, cwd=None, check=False):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "$ "
            + " ".join(map(str, cmd))
            + "\n\n"
            + p.stdout
        )

    return p


def git_repo(*args):
    p = run(
        ["git", *args],
        cwd=REPO,
        check=True,
    )
    return p.stdout.strip()


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            chunk = f.read(8 * 1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def atomic_json(path, obj):
    path = Path(path)
    tmp = Path(str(path) + ".tmp")

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write("\n")
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)


def command_info(name):
    path = shutil.which(name)

    if path is None:
        return {
            "available": False,
            "path": None,
            "version": None,
        }

    outputs = []

    for args in (
        [name, "--version"],
        [name, "-version"],
        [name, "-v"],
    ):

        p = run(args)

        text = (p.stdout or "").strip()

        if text:
            outputs.append(text[:3000])
            break

    return {
        "available": True,
        "path": path,
        "version": outputs[0] if outputs else None,
    }


# =============================================================================
# 2. GIT / SOURCE GATE
# =============================================================================

banner(
    "STAGE26-4B0 :: SCIENTIFIC ANCHOR"
)


git_repo(
    "fetch",
    "origin",
    "main",
)


head = git_repo(
    "rev-parse",
    "HEAD",
)

remote = git_repo(
    "rev-parse",
    "origin/main",
)

status = git_repo(
    "status",
    "--porcelain",
)


print("Expected HEAD :", EXPECTED_HEAD)
print("Local HEAD    :", head)
print("origin/main   :", remote)
print("Repo clean    :", status == "")


if head != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected local HEAD."
    )

if remote != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected origin/main."
    )

if status:
    raise RuntimeError(
        "Main repository must remain clean."
    )


# =============================================================================
# 3. STAGE26-4A RECEIPT / SOURCE
# =============================================================================

banner(
    "STAGE26-4B0 :: SOURCE RESTORATION GATE"
)


receipt_sha = sha256_file(
    SOURCE_RECEIPT
)


print(
    "4A receipt SHA256:"
)

print(
    " ",
    receipt_sha
)


if receipt_sha != EXPECTED_SOURCE_RECEIPT_SHA256:
    raise RuntimeError(
        "Stage26-4A receipt changed."
    )


source_receipt = json.loads(
    SOURCE_RECEIPT.read_text(
        encoding="utf-8"
    )
)


if not MONDAY_PCAP.exists():
    raise FileNotFoundError(
        MONDAY_PCAP
    )


resolved_pcap = MONDAY_PCAP.resolve(
    strict=True
)


pcap_size = int(
    resolved_pcap.stat().st_size
)


print(
    "PCAP resolved:"
)

print(
    " ",
    resolved_pcap
)

print(
    "Size:",
    pcap_size
)

print(
    "SHA recorded by verified 4A receipt:"
)

print(
    " ",
    source_receipt[
        "recovery"
    ][
        "verified_sha256"
    ]
)


if pcap_size != EXPECTED_PCAP_SIZE:
    raise RuntimeError(
        "PCAP size mismatch."
    )


if (
    source_receipt[
        "recovery"
    ][
        "verified_sha256"
    ]
    !=
    EXPECTED_PCAP_SHA256
):
    raise RuntimeError(
        "PCAP SHA receipt mismatch."
    )


# =============================================================================
# 4. STAGE20 RECONSTRUCTION PROVENANCE
# =============================================================================

banner(
    "STAGE26-4B0 :: HISTORICAL STAGE20 RECONSTRUCTION PROVENANCE"
)


geometry_sha = sha256_file(
    GEOMETRY_PROFILE
)


print(
    "Geometry profile SHA256:"
)

print(
    " ",
    geometry_sha
)


if geometry_sha != EXPECTED_GEOMETRY_SHA256:
    raise RuntimeError(
        "Geometry profile changed."
    )


geometry = json.loads(
    GEOMETRY_PROFILE.read_text(
        encoding="utf-8"
    )
)


reconstruction = geometry[
    "reconstruction_provenance"
]


print(
    json.dumps(
        reconstruction,
        indent=2,
        sort_keys=True,
    )
)


expected_lifecycle = (
    "CICFLOWMETER_FLOWGENERATOR_"
    +
    CICFLOWMETER_COMMIT.upper()
)


if (
    reconstruction[
        "flow_lifecycle_semantics_identifier"
    ]
    !=
    expected_lifecycle
):
    raise RuntimeError(
        "CICFlowMeter lifecycle commit mismatch."
    )


print(
    "\nParser semantics identifier:"
)

print(
    " ",
    reconstruction[
        "parser_semantics_identifier"
    ]
)


# =============================================================================
# 5. LOCAL RUNTIME TOOLCHAIN
# =============================================================================

banner(
    "STAGE26-4B0 :: RUNTIME TOOLCHAIN"
)


tool_names = [
    "java",
    "javac",
    "gradle",
    "mvn",
    "tshark",
    "editcap",
    "capinfos",
    "tcpdump",
    "gcc",
    "g++",
]


tool_inventory = {}


for tool in tool_names:

    info = command_info(
        tool
    )

    tool_inventory[
        tool
    ] = info


    print(
        f"{tool:10s}:",
        "FOUND"
        if info[
            "available"
        ]
        else
        "MISSING",
    )


    if info[
        "available"
    ]:

        print(
            "  path:",
            info[
                "path"
            ]
        )

        if info[
            "version"
        ]:

            for line in info[
                "version"
            ].splitlines()[
                :5
            ]:

                print(
                    "   ",
                    line
                )


# =============================================================================
# 6. NATIVE PCAP LIBRARY INVENTORY
# =============================================================================

banner(
    "STAGE26-4B0 :: NATIVE LIBRARIES"
)


native = run(
    [
        "bash",
        "-lc",
        "ldconfig -p 2>/dev/null | grep -iE 'libpcap|jnetpcap' || true",
    ]
).stdout.strip()


print(
    native
    if native
    else
    "<none>"
)


# =============================================================================
# 7. RESTORE EXACT CICFLOWMETER COMMIT
# =============================================================================

banner(
    "STAGE26-4B0 :: EXACT CICFLOWMETER CHECKOUT"
)


if CICFLOWMETER_DIR.exists():

    if not (
        CICFLOWMETER_DIR
        / ".git"
    ).exists():

        raise RuntimeError(
            "External CICFlowMeter directory is not a Git repository."
        )


    dirty = run(
        [
            "git",
            "status",
            "--porcelain",
        ],
        cwd=CICFLOWMETER_DIR,
        check=True,
    ).stdout.strip()


    if dirty:
        raise RuntimeError(
            "Existing CICFlowMeter checkout is dirty."
        )


else:

    run(
        [
            "git",
            "clone",
            "--no-checkout",
            CICFLOWMETER_REMOTE,
            str(
                CICFLOWMETER_DIR
            ),
        ],
        check=True,
    )


run(
    [
        "git",
        "fetch",
        "origin",
        CICFLOWMETER_COMMIT,
    ],
    cwd=CICFLOWMETER_DIR,
    check=True,
)


run(
    [
        "git",
        "checkout",
        "--detach",
        CICFLOWMETER_COMMIT,
    ],
    cwd=CICFLOWMETER_DIR,
    check=True,
)


cfm_head = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=CICFLOWMETER_DIR,
    check=True,
).stdout.strip()


cfm_tree = run(
    [
        "git",
        "rev-parse",
        "HEAD^{tree}",
    ],
    cwd=CICFLOWMETER_DIR,
    check=True,
).stdout.strip()


cfm_status = run(
    [
        "git",
        "status",
        "--porcelain",
    ],
    cwd=CICFLOWMETER_DIR,
    check=True,
).stdout.strip()


print(
    "Expected commit:",
    CICFLOWMETER_COMMIT
)

print(
    "Actual commit  :",
    cfm_head
)

print(
    "Tree SHA       :",
    cfm_tree
)

print(
    "Clean          :",
    cfm_status == ""
)


if cfm_head != CICFLOWMETER_COMMIT:
    raise RuntimeError(
        "Wrong CICFlowMeter commit."
    )


if cfm_status:
    raise RuntimeError(
        "CICFlowMeter checkout dirty."
    )


# =============================================================================
# 8. HASH KEY CICFLOWMETER FILES
# =============================================================================

banner(
    "STAGE26-4B0 :: CICFLOWMETER SOURCE HASHES"
)


key_files = [
    "src/main/java/cic/cs/unb/ca/ifm/CICFlowMeter.java",
    "src/main/java/cic/cs/unb/ca/jnetpcap/PacketReader.java",
    "src/main/java/cic/cs/unb/ca/jnetpcap/FlowGenerator.java",
    "src/main/java/cic/cs/unb/ca/jnetpcap/BasicPacketInfo.java",
    "src/main/java/cic/cs/unb/ca/jnetpcap/BasicFlow.java",
    "src/main/java/cic/cs/unb/ca/jnetpcap/FlowFeature.java",
]


cfm_file_hashes = {}


for relpath in key_files:

    path = (
        CICFLOWMETER_DIR
        / relpath
    )


    if path.exists():

        record = {
            "sha256":
                sha256_file(
                    path
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),
        }


        cfm_file_hashes[
            relpath
        ] = record


        print(
            record[
                "sha256"
            ],
            relpath
        )


    else:

        cfm_file_hashes[
            relpath
        ] = None


        print(
            "[MISSING]",
            relpath
        )


# =============================================================================
# 9. BUNDLED LIBRARIES / BUILD FILES
# =============================================================================

banner(
    "STAGE26-4B0 :: CICFLOWMETER LIBRARIES"
)


binary_records = []


for path in sorted(
    CICFLOWMETER_DIR.rglob(
        "*"
    )
):

    if not path.is_file():
        continue


    if path.suffix.lower() not in {
        ".jar",
        ".so",
        ".dll",
        ".dylib",
    }:
        continue


    record = {
        "path":
            str(
                path.relative_to(
                    CICFLOWMETER_DIR
                )
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


    binary_records.append(
        record
    )


    print(
        record[
            "sha256"
        ],
        record[
            "path"
        ]
    )


print(
    "\nBinary/library count:",
    len(
        binary_records
    )
)


banner(
    "STAGE26-4B0 :: BUILD FILES"
)


build_names = {
    "build.gradle",
    "build.gradle.kts",
    "settings.gradle",
    "settings.gradle.kts",
    "gradlew",
    "gradlew.bat",
    "pom.xml",
}


build_records = []


for path in sorted(
    CICFLOWMETER_DIR.rglob(
        "*"
    )
):

    if (
        path.is_file()
        and
        path.name in build_names
    ):

        record = {
            "path":
                str(
                    path.relative_to(
                        CICFLOWMETER_DIR
                    )
                ),

            "sha256":
                sha256_file(
                    path
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),
        }


        build_records.append(
            record
        )


        print(
            record[
                "sha256"
            ],
            record[
                "path"
            ]
        )


# =============================================================================
# 10. SEARCH OUR REPO FOR STAGE20 PARSER / PCAP CODE
# =============================================================================

banner(
    "STAGE26-4B0 :: LOCAL STAGE20 PARSER / PCAP SEARCH"
)


search_roots = [
    REPO
    / "scripts",

    REPO
    / "results"
    / "stage20_1c16_runtime_recovery",

    REPO
    / "results"
    / "stage20_1d_representation",
]


terms = [
    "STAGE20_FROZEN_IPV4_DIRECT_TRANSPORT_TCP6_UDP17_OTHER0",
    "CICFLOWMETER_FLOWGENERATOR",
    "FlowGenerator",
    "PacketReader",
    "source_faithful",
    "pcap",
]


hits = []


for root in search_roots:

    if not root.exists():
        continue


    for path in sorted(
        root.rglob(
            "*"
        )
    ):

        if not path.is_file():
            continue


        if path.suffix.lower() not in {
            ".py",
            ".json",
            ".md",
            ".txt",
            ".sh",
            ".java",
        }:
            continue


        if path.stat().st_size > (
            8 * 1024 * 1024
        ):
            continue


        text = path.read_text(
            encoding="utf-8",
            errors="ignore",
        )


        matched = [
            term
            for term in terms
            if term.lower()
            in text.lower()
        ]


        if matched:

            hit = {
                "path":
                    str(
                        path.relative_to(
                            REPO
                        )
                    ),

                "terms":
                    matched,

                "sha256":
                    sha256_file(
                        path
                    ),

                "size_bytes":
                    int(
                        path.stat().st_size
                    ),
            }


            hits.append(
                hit
            )


            print(
                hit[
                    "path"
                ]
            )

            print(
                "  terms:",
                ", ".join(
                    hit[
                        "terms"
                    ]
                )
            )

            print(
                "  SHA256:",
                hit[
                    "sha256"
                ]
            )


print(
    "\nMatching artifacts:",
    len(
        hits
    )
)


# =============================================================================
# 11. PCAP CONTAINER MAGIC ONLY
# =============================================================================

banner(
    "STAGE26-4B0 :: PCAP CONTAINER"
)


with resolved_pcap.open(
    "rb"
) as f:

    magic = f.read(
        4
    ).hex()


magic_map = {
    "d4c3b2a1":
        "PCAP_LITTLE_ENDIAN_MICROSECOND",

    "a1b2c3d4":
        "PCAP_BIG_ENDIAN_MICROSECOND",

    "4d3cb2a1":
        "PCAP_LITTLE_ENDIAN_NANOSECOND",

    "a1b23c4d":
        "PCAP_BIG_ENDIAN_NANOSECOND",

    "0a0d0d0a":
        "PCAPNG",
}


pcap_type = magic_map.get(
    magic,
    "UNKNOWN",
)


print(
    "Magic:",
    magic
)

print(
    "Type :",
    pcap_type
)


if pcap_type == "UNKNOWN":
    raise RuntimeError(
        "Unknown PCAP format."
    )


# =============================================================================
# 12. WRITE INVENTORY RECEIPT
# =============================================================================

inventory = {
    "schema":
        "stage26_4b0_extractor_runtime_inventory_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4B0",

    "status":
        "INVENTORY_COMPLETE_NO_EXTRACTION",

    "parent_commit":
        EXPECTED_HEAD,

    "stage26_4a_receipt_sha256":
        receipt_sha,

    "pcap": {
        "path":
            str(
                MONDAY_PCAP
            ),

        "resolved_path":
            str(
                resolved_pcap
            ),

        "size_bytes":
            pcap_size,

        "sha256":
            EXPECTED_PCAP_SHA256,

        "container_magic":
            magic,

        "container_type":
            pcap_type,
    },

    "stage20_reconstruction_provenance":
        reconstruction,

    "runtime_tools":
        tool_inventory,

    "native_pcap_libraries":
        native,

    "cicflowmeter": {
        "remote":
            CICFLOWMETER_REMOTE,

        "commit":
            cfm_head,

        "tree":
            cfm_tree,

        "key_files":
            cfm_file_hashes,

        "binary_libraries":
            binary_records,

        "build_files":
            build_records,
    },

    "local_stage20_candidate_artifacts":
        hits,

    "scientific_boundary": {
        "pcap_packets_iterated":
            False,

        "flow_reconstruction":
            False,

        "feature_extraction":
            False,

        "timing":
            False,

        "model_loading":
            False,

        "inference":
            False,

        "labels":
            False,

        "gpu":
            False,

        "main_repo_modified":
            False,
    },

    "next_action":
        "FREEZE_STAGE26_4B_EXTRACTION_PROTOCOL",
}


atomic_json(
    INVENTORY_RECEIPT,
    inventory,
)


inventory_sha = sha256_file(
    INVENTORY_RECEIPT
)


# =============================================================================
# 13. FINAL AUDIT
# =============================================================================

banner(
    "STAGE26-4B0 :: FINAL AUDIT"
)


final_head = git_repo(
    "rev-parse",
    "HEAD",
)

final_remote = git_repo(
    "rev-parse",
    "origin/main",
)

final_status = git_repo(
    "status",
    "--porcelain",
)


print(
    "Local HEAD :",
    final_head
)

print(
    "Remote HEAD:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_HEAD:
    raise RuntimeError(
        "Main repository HEAD changed."
    )

if final_remote != EXPECTED_HEAD:
    raise RuntimeError(
        "origin/main changed."
    )

if final_status:
    raise RuntimeError(
        "Main repository became dirty."
    )


banner(
    "STAGE26-4B0 INVENTORY COMPLETE"
)


print(
    "CICFLOWMETER COMMIT:"
)

print(
    " ",
    cfm_head
)


print(
    "\nCICFLOWMETER TREE:"
)

print(
    " ",
    cfm_tree
)


print(
    "\nPCAP TYPE:"
)

print(
    " ",
    pcap_type
)


print(
    "\nLOCAL STAGE20 CANDIDATES:"
)

print(
    " ",
    len(
        hits
    )
)


print(
    "\nINVENTORY RECEIPT:"
)

print(
    " ",
    INVENTORY_RECEIPT
)

print(
    " SHA256:",
    inventory_sha
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  EXACT MONDAY SOURCE         : RESTORED"
)

print(
    "  CICFLOWMETER COMMIT         : IDENTIFIED"
)

print(
    "  RUNTIME                     : INVENTORIED"
)

print(
    "  STAGE20 PARSER CANDIDATES   : SEARCHED"
)

print(
    "  PACKETS ITERATED            : NO"
)

print(
    "  EXTRACTION                  : NO"
)

print(
    "  EXTRACTION TIMING           : NO"
)

print(
    "  GPU                         : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Freeze Stage26-4B extraction protocol before first benchmark."
)


STAGE26-4B0 :: SCIENTIFIC ANCHOR
Expected HEAD : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Local HEAD    : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
origin/main   : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Repo clean    : True

STAGE26-4B0 :: SOURCE RESTORATION GATE
4A receipt SHA256:
  9a6fdb032e7be7e047a66cf08b5d59ee5523480eb20faf6975403fafe4a91b0a
PCAP resolved:
  /kaggle/working/stage26_hf_cache/datasets--bvsam--cic-ids-2017/blobs/f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972
Size: 10822507416
SHA recorded by verified 4A receipt:
  f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972

STAGE26-4B0 :: HISTORICAL STAGE20 RECONSTRUCTION PROVENANCE
Geometry profile SHA256:
  3a26d6499334c12ea4e9272aef4250761c6cf4399e7fb4d33ef236f12d0b7272
{
  "captured_ipv4_length_definition": "captured frame bytes beginning at IPv4 header byte 0 through the end of the captured frame; Ethernet/VLAN bytes excluded",
  "exportable_flow_definition": "historical finished flows 

In [7]:
# =============================================================================
# STAGE26-4A-RECOVERY-B
# RESUME EXACT MONDAY PCAP DOWNLOAD WITH XET DISABLED
#
# EXISTING STATE:
#   partial .incomplete blob ≈ 8.403 GiB / 83.37%
#
# STRATEGY:
#   - DO NOT delete partial cache.
#   - Spawn a fresh Python process.
#   - Set HF_HUB_DISABLE_XET=1 BEFORE huggingface_hub import.
#   - Set HF_HUB_DOWNLOAD_TIMEOUT=300.
#   - Reuse exact same Hugging Face cache.
#   - Verify final size + SHA256.
#
# NO:
#   - packet parsing
#   - flow reconstruction
#   - extraction timing
#   - model inference
#   - labels
#   - GPU
#   - Git modification
# =============================================================================

from __future__ import annotations

import os
import sys
import json
import time
import hashlib
import shutil
import subprocess
import textwrap
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

SOURCE_ROOT = (
    STAGE26_ROOT
    / "sources"
)

EXTRACTION_ROOT = (
    STAGE26_ROOT
    / "extraction"
)

SOURCE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


EXPECTED_HEAD = (
    "347d93f21d454cc5bda2c45889c890c67cdf0ecc"
)

HF_CACHE = Path(
    "/kaggle/working/stage26_hf_cache"
)

HF_REPO_ID = (
    "bvsam/cic-ids-2017"
)

HF_REVISION = (
    "e810c1cc98270ec271a1df917b9de0786c33f343"
)

HF_FILENAME = (
    "pcap/Monday-WorkingHours.pcap"
)

SOURCE_BASENAME = (
    "Monday-WorkingHours.pcap"
)

EXPECTED_SIZE = (
    10_822_507_416
)

EXPECTED_SHA = (
    "f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972"
)

EXPECTED_PACKET_COUNT = (
    11_709_971
)

EXPECTED_VALID_IPV4_COUNT = (
    11_626_492
)

EXPECTED_EXPORTABLE_FLOW_COUNT = (
    529_601
)


PARTIAL_PATH = (
    HF_CACHE
    / "datasets--bvsam--cic-ids-2017"
    / "blobs"
    / (
        EXPECTED_SHA
        + ".incomplete"
    )
)

CANONICAL_SOURCE = (
    SOURCE_ROOT
    / SOURCE_BASENAME
)

SOURCE_RECEIPT = (
    EXTRACTION_ROOT
    / "stage26_4a_monday_source_restoration_receipt.json"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 112
    )

    print(text)

    print(
        "=" * 112
    )


def human(n):

    n = float(n)

    for unit in [
        "B",
        "KiB",
        "MiB",
        "GiB",
        "TiB",
    ]:

        if n < 1024 or unit == "TiB":

            return f"{n:.3f} {unit}"

        n /= 1024


def sha256_file(
    path,
    *,
    progress=False,
):

    path = Path(path)

    h = hashlib.sha256()

    total = 0

    next_report = (
        2 * 1024**3
    )


    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                16 * 1024 * 1024
            )

            if not chunk:
                break


            h.update(
                chunk
            )

            total += len(
                chunk
            )


            if (
                progress
                and
                total >= next_report
            ):

                print(
                    f"  hashed "
                    f"{total / 1024**3:.2f} GiB",
                    flush=True,
                )

                next_report += (
                    2 * 1024**3
                )


    return h.hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(path)

    tmp = Path(
        str(path)
        + ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )


    os.replace(
        tmp,
        path,
    )


def git(*args):

    p = subprocess.run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


# =============================================================================
# 2. SCIENTIFIC GIT ANCHOR
# =============================================================================

banner(
    "STAGE26-4A-RECOVERY-B :: GIT ANCHOR"
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD:",
    EXPECTED_HEAD
)

print(
    "Local HEAD   :",
    head
)

print(
    "origin/main  :",
    remote
)

print(
    "Repo clean   :",
    status == ""
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected local HEAD."
    )


if remote != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected origin/main."
    )


if status:

    raise RuntimeError(
        "Repository must remain clean."
    )


# =============================================================================
# 3. PARTIAL CACHE AUDIT
# =============================================================================

banner(
    "STAGE26-4A-RECOVERY-B :: PARTIAL CACHE AUDIT"
)


print(
    "Partial path:"
)

print(
    " ",
    PARTIAL_PATH
)


if PARTIAL_PATH.exists():

    partial_size_before = int(
        PARTIAL_PATH.stat().st_size
    )


    print(
        "Partial exists: YES"
    )

    print(
        "Partial size  :",
        human(
            partial_size_before
        )
    )

    print(
        "Completion    :",
        f"{partial_size_before / EXPECTED_SIZE:.2%}"
    )


else:

    partial_size_before = 0


    print(
        "Partial exists: NO"
    )

    print(
        "The retry will still use the exact frozen source."
    )


disk_before = shutil.disk_usage(
    "/kaggle/working"
)


print(
    "\nWorkspace free:",
    human(
        disk_before.free
    )
)


remaining_bytes = max(
    0,
    EXPECTED_SIZE
    -
    partial_size_before,
)


print(
    "Remaining source bytes:",
    human(
        remaining_bytes
    )
)


# =============================================================================
# 4. CREATE FRESH-SUBPROCESS DOWNLOADER
# =============================================================================

banner(
    "STAGE26-4A-RECOVERY-B :: HTTP FALLBACK / RESUME"
)


downloader_script = (
    STAGE26_ROOT
    / "runtime"
    / "stage26_4a_http_resume.py"
)

downloader_script.parent.mkdir(
    parents=True,
    exist_ok=True,
)


script = r'''
import os

# MUST be set before importing huggingface_hub.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"

from pathlib import Path
from huggingface_hub import hf_hub_download


CACHE = "/kaggle/working/stage26_hf_cache"

print(
    "Fresh subprocess environment:",
    flush=True,
)

print(
    "  HF_HUB_DISABLE_XET      =",
    os.environ.get("HF_HUB_DISABLE_XET"),
    flush=True,
)

print(
    "  HF_HUB_DOWNLOAD_TIMEOUT =",
    os.environ.get("HF_HUB_DOWNLOAD_TIMEOUT"),
    flush=True,
)

print(
    "  HF_HUB_ETAG_TIMEOUT     =",
    os.environ.get("HF_HUB_ETAG_TIMEOUT"),
    flush=True,
)

print(
    "\nCalling hf_hub_download using existing cache...",
    flush=True,
)


path = hf_hub_download(
    repo_id="bvsam/cic-ids-2017",
    repo_type="dataset",
    filename="pcap/Monday-WorkingHours.pcap",
    revision="e810c1cc98270ec271a1df917b9de0786c33f343",
    cache_dir=CACHE,
    local_files_only=False,
    force_download=False,
)


resolved = Path(
    path
).resolve(
    strict=True
)


print(
    "\nDOWNLOAD_RETURNED",
    flush=True,
)

print(
    str(resolved),
    flush=True,
)

print(
    "SIZE_BYTES",
    resolved.stat().st_size,
    flush=True,
)
'''


downloader_script.write_text(
    textwrap.dedent(
        script
    ).lstrip(),
    encoding="utf-8",
)


# =============================================================================
# 5. RUN FRESH PROCESS WITH STREAMED OUTPUT
# =============================================================================

env = os.environ.copy()

# Also set at process level for defense in depth.
env[
    "HF_HUB_DISABLE_XET"
] = "1"

env[
    "HF_HUB_DOWNLOAD_TIMEOUT"
] = "300"

env[
    "HF_HUB_ETAG_TIMEOUT"
] = "60"


print(
    "Launching fresh Python subprocess..."
)

print(
    "Existing partial file is NOT deleted."
)

print()


proc = subprocess.Popen(
    [
        sys.executable,
        "-u",
        str(
            downloader_script
        ),
    ],
    cwd=REPO,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)


captured_lines = []


while True:

    line = proc.stdout.readline()

    if line:

        print(
            line,
            end="",
            flush=True,
        )

        captured_lines.append(
            line
        )


    if (
        proc.poll() is not None
        and
        not line
    ):

        break


returncode = proc.wait()


print(
    "\nSubprocess return code:",
    returncode
)


if returncode != 0:

    current_partial_size = (
        int(
            PARTIAL_PATH.stat().st_size
        )
        if PARTIAL_PATH.exists()
        else 0
    )


    print(
        "\nCurrent partial size:",
        human(
            current_partial_size
        )
    )


    print(
        "Current completion:",
        f"{current_partial_size / EXPECTED_SIZE:.2%}"
        if EXPECTED_SIZE
        else "N/A",
    )


    raise RuntimeError(
        "HTTP/Xet-disabled resume did not complete.\n"
        "DO NOT delete the partial cache; paste this full output."
    )


# =============================================================================
# 6. OFFLINE-RESOLVE COMPLETED CACHE ENTRY
# =============================================================================

banner(
    "STAGE26-4A-RECOVERY-B :: OFFLINE COMPLETION GATE"
)


# Use another fresh subprocess with network disabled semantically via
# local_files_only=True. This verifies Hugging Face now considers it complete.
offline_script = (
    STAGE26_ROOT
    / "runtime"
    / "stage26_4a_offline_resolve.py"
)


offline_script.write_text(
    r'''
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

from pathlib import Path
from huggingface_hub import hf_hub_download


path = hf_hub_download(
    repo_id="bvsam/cic-ids-2017",
    repo_type="dataset",
    filename="pcap/Monday-WorkingHours.pcap",
    revision="e810c1cc98270ec271a1df917b9de0786c33f343",
    cache_dir="/kaggle/working/stage26_hf_cache",
    local_files_only=True,
)


resolved = Path(path).resolve(strict=True)

print(str(resolved))
print(resolved.stat().st_size)
'''.lstrip(),
    encoding="utf-8",
)


offline = subprocess.run(
    [
        sys.executable,
        str(
            offline_script
        ),
    ],
    cwd=REPO,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)


print(
    offline.stdout
)


if offline.returncode != 0:

    raise RuntimeError(
        "Download process returned success but Hugging Face "
        "does not consider the cache entry complete."
    )


offline_lines = [
    x.strip()
    for x in offline.stdout.splitlines()
    if x.strip()
]


if len(
    offline_lines
) < 2:

    raise RuntimeError(
        "Unable to resolve completed cached source."
    )


completed_path = Path(
    offline_lines[
        -2
    ]
).resolve(
    strict=True
)


completed_size = int(
    offline_lines[
        -1
    ]
)


print(
    "Completed object:"
)

print(
    " ",
    completed_path
)

print(
    "Size:"
)

print(
    " ",
    human(
        completed_size
    )
)


if completed_size != EXPECTED_SIZE:

    raise RuntimeError(
        "Completed PCAP size mismatch."
    )


# =============================================================================
# 7. FULL SCIENTIFIC BYTE HASH
# =============================================================================

banner(
    "STAGE26-4A-RECOVERY-B :: FULL SHA256 GATE"
)


print(
    "Expected size:"
)

print(
    " ",
    EXPECTED_SIZE
)


print(
    "Expected SHA256:"
)

print(
    " ",
    EXPECTED_SHA
)


print(
    "\nHashing complete 10.08 GiB source..."
)


actual_sha = sha256_file(
    completed_path,
    progress=True,
)


print(
    "\nActual SHA256:"
)

print(
    " ",
    actual_sha
)


if actual_sha != EXPECTED_SHA:

    raise RuntimeError(
        "Completed source SHA256 mismatch."
    )


print(
    "\n[PASS] Exact historical Monday bytes restored."
)


# =============================================================================
# 8. CANONICAL STAGE26 SOURCE POINTER
# =============================================================================

banner(
    "STAGE26-4A-RECOVERY-B :: CANONICAL SOURCE"
)


if (
    CANONICAL_SOURCE.exists()
    or
    CANONICAL_SOURCE.is_symlink()
):

    CANONICAL_SOURCE.unlink()


CANONICAL_SOURCE.symlink_to(
    completed_path
)


resolved = CANONICAL_SOURCE.resolve(
    strict=True
)


if resolved != completed_path:

    raise RuntimeError(
        "Canonical source pointer mismatch."
    )


print(
    "Canonical source:"
)

print(
    " ",
    CANONICAL_SOURCE
)

print(
    "Resolves to:"
)

print(
    " ",
    resolved
)


# =============================================================================
# 9. STAGE26-4A RESTORATION RECEIPT
# =============================================================================

receipt = {
    "schema":
        "stage26_4a_monday_source_restoration_receipt_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-4A",

    "status":
        "PASS_EXACT_SOURCE_RESTORED",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_HEAD,

    "source_identity": {
        "repo_id":
            HF_REPO_ID,

        "revision":
            HF_REVISION,

        "filename":
            HF_FILENAME,

        "size_bytes":
            EXPECTED_SIZE,

        "sha256":
            EXPECTED_SHA,

        "historical_raw_packet_count":
            EXPECTED_PACKET_COUNT,

        "historical_valid_ipv4_packet_count":
            EXPECTED_VALID_IPV4_COUNT,

        "historical_exportable_flow_count":
            EXPECTED_EXPORTABLE_FLOW_COUNT,
    },

    "recovery": {
        "prior_xet_transfer_stalled":
            True,

        "partial_blob_preserved":
            True,

        "partial_size_before_retry":
            partial_size_before,

        "partial_fraction_before_retry":
            (
                partial_size_before
                /
                EXPECTED_SIZE
            ),

        "retry_method":
            "HF_HUB_DISABLE_XET_HTTP_FALLBACK_IN_FRESH_SUBPROCESS",

        "HF_HUB_DOWNLOAD_TIMEOUT":
            300,

        "HF_HUB_ETAG_TIMEOUT":
            60,

        "completed_cache_path":
            str(
                completed_path
            ),

        "canonical_stage26_path":
            str(
                CANONICAL_SOURCE
            ),

        "verified_size_bytes":
            completed_size,

        "verified_sha256":
            actual_sha,
    },

    "scientific_boundary": {
        "packet_iteration":
            False,

        "packet_parsing":
            False,

        "flow_reconstruction":
            False,

        "extraction_timing":
            False,

        "throughput_computed":
            False,

        "model_loaded":
            False,

        "inference":
            False,

        "labels_read":
            False,

        "Thursday_accessed":
            False,

        "Friday_accessed":
            False,

        "gpu_used":
            False,

        "git_modified":
            False,
    },

    "next_action":
        (
            "STAGE26-4B_FREEZE_EXTRACTION_SUBPROTOCOL_"
            "BEFORE_FIRST_EXTRACTION_TIMING"
        ),
}


atomic_json(
    SOURCE_RECEIPT,
    receipt,
)


receipt_sha = sha256_file(
    SOURCE_RECEIPT
)


# =============================================================================
# 10. FINAL AUDIT
# =============================================================================

banner(
    "STAGE26-4A-RECOVERY-B :: FINAL AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


disk_after = shutil.disk_usage(
    "/kaggle/working"
)


print(
    "Local HEAD :",
    final_head
)

print(
    "Remote HEAD:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


print(
    "\nWorkspace free:"
)

print(
    " ",
    human(
        disk_after.free
    )
)


if final_head != EXPECTED_HEAD:

    raise RuntimeError(
        "HEAD changed unexpectedly."
    )


if final_remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed unexpectedly."
    )


if final_status:

    raise RuntimeError(
        "Git repository changed unexpectedly."
    )


banner(
    "STAGE26-4A SOURCE RESTORATION COMPLETE"
)


print(
    "SOURCE:"
)

print(
    " ",
    SOURCE_BASENAME
)


print(
    "\nSIZE:"
)

print(
    " ",
    EXPECTED_SIZE
)


print(
    "\nSHA256:"
)

print(
    " ",
    actual_sha
)


print(
    "\nHISTORICAL PACKETS:"
)

print(
    " ",
    EXPECTED_PACKET_COUNT
)


print(
    "\nSOURCE RECEIPT:"
)

print(
    " ",
    SOURCE_RECEIPT
)

print(
    " SHA256:",
    receipt_sha
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  MONDAY PCAP RESTORED       : PASS"
)

print(
    "  BYTE IDENTITY              : PASS"
)

print(
    "  PACKETS PARSED             : NO"
)

print(
    "  EXTRACTION TIMING          : NO"
)

print(
    "  MODELS LOADED              : NO"
)

print(
    "  HOLDOUT ACCESSED           : NO"
)

print(
    "  GPU USED                   : NO"
)


print(
    "\nNEXT:"
)

print(
    "  STAGE26-4B — freeze the bounded PCAP extraction sample"
)

print(
    "  and all I/O/startup/serialization/extractor semantics"
)

print(
    "  BEFORE the first timed extraction measurement."
)


STAGE26-4A-RECOVERY-B :: GIT ANCHOR
Expected HEAD: 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Local HEAD   : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
origin/main  : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Repo clean   : True

STAGE26-4A-RECOVERY-B :: PARTIAL CACHE AUDIT
Partial path:
  /kaggle/working/stage26_hf_cache/datasets--bvsam--cic-ids-2017/blobs/f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972.incomplete
Partial exists: NO
The retry will still use the exact frozen source.

Workspace free: 7.141 GiB
Remaining source bytes: 10.079 GiB

STAGE26-4A-RECOVERY-B :: HTTP FALLBACK / RESUME
Launching fresh Python subprocess...
Existing partial file is NOT deleted.

Fresh subprocess environment:
  HF_HUB_DISABLE_XET      = 1
  HF_HUB_DOWNLOAD_TIMEOUT = 300
  HF_HUB_ETAG_TIMEOUT     = 60

Calling hf_hub_download using existing cache...

DOWNLOAD_RETURNED
/kaggle/working/stage26_hf_cache/datasets--bvsam--cic-ids-2017/blobs/f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc

In [8]:
# =============================================================================
# STAGE26 — GITHUB RELEASE CORPUS INVENTORY
#
# PURPOSE:
#   Inspect ALL GitHub Releases and ALL release assets before Stage26-4B.
#
# HARD SCIENTIFIC RULE ADDED TO OUR WORKFLOW:
#   DO NOT RECREATE ANY EXISTING CORPUS FROM THE MONDAY PCAP.
#
# THIS CELL:
#   - lists every GitHub Release;
#   - lists every release asset;
#   - records asset name, size, digest, download count, dates;
#   - highlights likely corpus/packet/flow/manifest assets;
#   - DOES NOT DOWNLOAD any release asset;
#   - DOES NOT open/parse the PCAP;
#   - DOES NOT perform extraction;
#   - DOES NOT modify Git.
# =============================================================================

from __future__ import annotations

import json
import urllib.request
import urllib.error
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. CONFIG
# =============================================================================

OWNER = "themubasshir"
REPO_NAME = "ids2018-validation-safe-ablation"

API_URL = (
    f"https://api.github.com/repos/"
    f"{OWNER}/{REPO_NAME}/releases?per_page=100&page=1"
)

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

OUT = Path(
    "/kaggle/working/stage26_deployment_profiling/"
    "extraction/"
    "stage26_github_release_corpus_inventory.json"
)

EXPECTED_HEAD = (
    "347d93f21d454cc5bda2c45889c890c67cdf0ecc"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 118
    )

    print(text)

    print(
        "=" * 118
    )


def human_bytes(n):

    n = float(
        n
    )

    for unit in [
        "B",
        "KiB",
        "MiB",
        "GiB",
        "TiB",
    ]:

        if (
            n < 1024
            or
            unit == "TiB"
        ):

            return (
                f"{n:.3f} {unit}"
            )

        n /= 1024


def git(*args):

    import subprocess

    p = subprocess.run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


# =============================================================================
# 2. GIT ANCHOR
# =============================================================================

banner(
    "GITHUB RELEASE CORPUS AUDIT :: SCIENTIFIC ANCHOR"
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD :",
    EXPECTED_HEAD
)

print(
    "Local HEAD    :",
    head
)

print(
    "origin/main   :",
    remote
)

print(
    "Repo clean    :",
    status == ""
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected local HEAD."
    )


if remote != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected origin/main."
    )


if status:

    raise RuntimeError(
        "Repository must remain clean."
    )


# =============================================================================
# 3. OPTIONAL AUTHENTICATION
# =============================================================================

banner(
    "GITHUB RELEASE CORPUS AUDIT :: API AUTH"
)


headers = {
    "Accept":
        "application/vnd.github+json",

    "X-GitHub-Api-Version":
        "2022-11-28",

    "User-Agent":
        "stage26-release-corpus-audit",
}


token_present = False


try:

    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret(
        "GITHUB_TOKEN"
    )

    if token:

        headers[
            "Authorization"
        ] = (
            f"Bearer {token}"
        )

        token_present = True

        print(
            "[FOUND] GITHUB_TOKEN; using authenticated read-only API request."
        )

        del token


except Exception as exc:

    print(
        "[INFO] GITHUB_TOKEN unavailable; public API access will be used."
    )

    print(
        " ",
        type(exc).__name__,
        str(exc),
    )


# =============================================================================
# 4. FETCH RELEASE METADATA ONLY
# =============================================================================

banner(
    "GITHUB RELEASE CORPUS AUDIT :: RELEASE LIST"
)


request = urllib.request.Request(
    API_URL,
    headers=headers,
    method="GET",
)


try:

    with urllib.request.urlopen(
        request,
        timeout=60,
    ) as response:

        raw = response.read()

        status_code = response.status


except urllib.error.HTTPError as exc:

    body = exc.read().decode(
        "utf-8",
        errors="replace",
    )

    raise RuntimeError(
        f"GitHub API HTTP {exc.code}\n{body}"
    )


except Exception as exc:

    raise RuntimeError(
        "Could not query GitHub Releases API."
    ) from exc


if status_code != 200:

    raise RuntimeError(
        f"Unexpected GitHub API status: {status_code}"
    )


releases = json.loads(
    raw.decode(
        "utf-8"
    )
)


if not isinstance(
    releases,
    list,
):

    raise RuntimeError(
        "GitHub Releases API did not return a list."
    )


print(
    "Published/draft-visible releases returned:",
    len(
        releases
    )
)


# =============================================================================
# 5. INVENTORY EVERY RELEASE ASSET
# =============================================================================

keywords = [
    "corpus",
    "packet",
    "packets",
    "flow",
    "flows",
    "stage20",
    "monday",
    "tuesday",
    "wednesday",
    "thursday",
    "friday",
    "train",
    "validation",
    "test",
    "manifest",
    "offset",
    "length",
    "index",
    "geometry",
    "representation",
    "parquet",
    "npz",
    "npy",
    "zip",
    "tar",
]


release_records = []

all_assets = []

candidate_assets = []


for release in releases:

    tag = release.get(
        "tag_name"
    )

    name = release.get(
        "name"
    )

    release_id = release.get(
        "id"
    )

    assets = release.get(
        "assets",
        []
    )


    record = {
        "id":
            release_id,

        "tag_name":
            tag,

        "name":
            name,

        "draft":
            release.get(
                "draft"
            ),

        "prerelease":
            release.get(
                "prerelease"
            ),

        "created_at":
            release.get(
                "created_at"
            ),

        "published_at":
            release.get(
                "published_at"
            ),

        "target_commitish":
            release.get(
                "target_commitish"
            ),

        "asset_count":
            len(
                assets
            ),

        "assets":
            [],
    }


    print(
        "\n"
        + "-" * 118
    )

    print(
        f"RELEASE: {name!r}"
    )

    print(
        f"TAG    : {tag}"
    )

    print(
        f"ID     : {release_id}"
    )

    print(
        f"DRAFT  : {release.get('draft')}"
    )

    print(
        f"PRE    : {release.get('prerelease')}"
    )

    print(
        f"ASSETS : {len(assets)}"
    )


    if not assets:

        print(
            "  <no attached release assets>"
        )


    for asset in assets:

        asset_name = asset.get(
            "name",
            "",
        )

        asset_size = int(
            asset.get(
                "size",
                0,
            )
            or 0
        )

        digest = asset.get(
            "digest"
        )


        lower = asset_name.lower()


        matched_keywords = [
            kw
            for kw in keywords
            if kw in lower
        ]


        asset_record = {
            "release_id":
                release_id,

            "release_tag":
                tag,

            "release_name":
                name,

            "asset_id":
                asset.get(
                    "id"
                ),

            "name":
                asset_name,

            "label":
                asset.get(
                    "label"
                ),

            "content_type":
                asset.get(
                    "content_type"
                ),

            "state":
                asset.get(
                    "state"
                ),

            "size_bytes":
                asset_size,

            "size_human":
                human_bytes(
                    asset_size
                ),

            "digest":
                digest,

            "download_count":
                asset.get(
                    "download_count"
                ),

            "created_at":
                asset.get(
                    "created_at"
                ),

            "updated_at":
                asset.get(
                    "updated_at"
                ),

            "browser_download_url":
                asset.get(
                    "browser_download_url"
                ),

            "matched_keywords":
                matched_keywords,
        }


        record[
            "assets"
        ].append(
            asset_record
        )

        all_assets.append(
            asset_record
        )


        if matched_keywords:

            candidate_assets.append(
                asset_record
            )


        marker = (
            "***"
            if matched_keywords
            else
            "   "
        )


        print(
            f"{marker} "
            f"{asset_name}"
        )

        print(
            f"      size    : "
            f"{human_bytes(asset_size)} "
            f"({asset_size:,} bytes)"
        )

        print(
            f"      digest  : "
            f"{digest}"
        )

        print(
            f"      type    : "
            f"{asset.get('content_type')}"
        )

        print(
            f"      downloads: "
            f"{asset.get('download_count')}"
        )


        if matched_keywords:

            print(
                "      matched :",
                ", ".join(
                    matched_keywords
                )
            )


    release_records.append(
        record
    )


# =============================================================================
# 6. GLOBAL SUMMARY
# =============================================================================

banner(
    "GITHUB RELEASE CORPUS AUDIT :: GLOBAL ASSET SUMMARY"
)


total_asset_bytes = sum(
    x[
        "size_bytes"
    ]
    for x in all_assets
)


print(
    "Release count:",
    len(
        release_records
    )
)

print(
    "Asset count:",
    len(
        all_assets
    )
)

print(
    "Total attached asset bytes:",
    f"{total_asset_bytes:,}"
)

print(
    "Total attached asset size:",
    human_bytes(
        total_asset_bytes
    )
)

print(
    "Corpus/packet/flow candidate assets:",
    len(
        candidate_assets
    )
)


# =============================================================================
# 7. HIGHLIGHT LIKELY CORPUS ASSETS
# =============================================================================

banner(
    "GITHUB RELEASE CORPUS AUDIT :: LIKELY CORPUS / REPRESENTATION ASSETS"
)


if not candidate_assets:

    print(
        "No asset names matched the corpus/flow/packet keywords."
    )


else:

    for i, asset in enumerate(
        candidate_assets,
        start=1,
    ):

        print(
            f"[{i:02d}] "
            f"{asset['name']}"
        )

        print(
            "     release:",
            asset[
                "release_name"
            ]
        )

        print(
            "     tag    :",
            asset[
                "release_tag"
            ]
        )

        print(
            "     size   :",
            asset[
                "size_human"
            ]
        )

        print(
            "     digest :",
            asset[
                "digest"
            ]
        )

        print(
            "     match  :",
            ", ".join(
                asset[
                    "matched_keywords"
                ]
            )
        )


# =============================================================================
# 8. DAY COVERAGE HEURISTIC
# =============================================================================

banner(
    "GITHUB RELEASE CORPUS AUDIT :: DAY / CORPUS COVERAGE"
)


day_names = [
    "monday",
    "tuesday",
    "wednesday",
    "thursday",
    "friday",
]


day_coverage = {}


for day in day_names:

    matching = [
        x
        for x in all_assets
        if day
        in x[
            "name"
        ].lower()
    ]

    day_coverage[
        day
    ] = [
        {
            "release_tag":
                x[
                    "release_tag"
                ],

            "asset_name":
                x[
                    "name"
                ],

            "size_bytes":
                x[
                    "size_bytes"
                ],

            "digest":
                x[
                    "digest"
                ],
        }
        for x in matching
    ]


    print(
        f"{day.upper():10s}: "
        f"{len(matching)} matching assets"
    )


    for asset in matching:

        print(
            "   ",
            asset[
                "name"
            ],
            "|",
            asset[
                "size_human"
            ],
        )


# =============================================================================
# 9. WRITE LOCAL AUDIT RECEIPT
# =============================================================================

inventory = {
    "schema":
        "stage26_github_release_corpus_inventory_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "repository":
        f"{OWNER}/{REPO_NAME}",

    "scientific_parent":
        EXPECTED_HEAD,

    "api_authenticated":
        token_present,

    "release_count":
        len(
            release_records
        ),

    "asset_count":
        len(
            all_assets
        ),

    "total_asset_bytes":
        total_asset_bytes,

    "candidate_asset_count":
        len(
            candidate_assets
        ),

    "releases":
        release_records,

    "likely_corpus_assets":
        candidate_assets,

    "day_coverage":
        day_coverage,

    "scientific_policy": {
        "recreate_existing_corpus_from_pcap":
            False,

        "pcap_role":
            (
                "RAW EXTRACTION BENCHMARK SOURCE ONLY; "
                "NOT A CORPUS REGENERATION SOURCE"
            ),

        "release_corpus_role":
            (
                "AUTHORITATIVE DOWNSTREAM CORPUS WHERE "
                "RELEASE ASSETS PROVIDE THE REQUIRED DATA"
            ),

        "release_asset_download_performed":
            False,

        "pcap_opened":
            False,

        "pcap_packets_iterated":
            False,

        "extraction_performed":
            False,

        "timing_performed":
            False,

        "model_inference":
            False,

        "gpu_used":
            False,

        "git_modified":
            False,
    },
}


OUT.parent.mkdir(
    parents=True,
    exist_ok=True,
)


OUT.write_text(
    json.dumps(
        inventory,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


# =============================================================================
# 10. FINAL GIT AUDIT
# =============================================================================

banner(
    "GITHUB RELEASE CORPUS AUDIT :: FINAL SCIENTIFIC STATE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "Local HEAD :",
    final_head
)

print(
    "Remote HEAD:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_HEAD:

    raise RuntimeError(
        "HEAD changed unexpectedly."
    )


if final_remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed unexpectedly."
    )


if final_status:

    raise RuntimeError(
        "Repository changed unexpectedly."
    )


print(
    "\nInventory:"
)

print(
    " ",
    OUT
)


print(
    "\nHARD POLICY:"
)

print(
    "  RECREATE RELEASE CORPUS FROM PCAP : NO"
)

print(
    "  DOWNLOAD RELEASE ASSETS NOW       : NO"
)

print(
    "  PCAP EXTRACTION                    : NO"
)

print(
    "  EXTRACTION TIMING                  : NO"
)

print(
    "  GPU                                : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Inspect exact Release assets and map them to Stage26"
)

print(
    "  extraction / representation / end-to-end boundaries BEFORE"
)

print(
    "  freezing Stage26-4B."
)


GITHUB RELEASE CORPUS AUDIT :: SCIENTIFIC ANCHOR
Expected HEAD : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Local HEAD    : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
origin/main   : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Repo clean    : True

GITHUB RELEASE CORPUS AUDIT :: API AUTH
[FOUND] GITHUB_TOKEN; using authenticated read-only API request.

GITHUB RELEASE CORPUS AUDIT :: RELEASE LIST
Published/draft-visible releases returned: 2

----------------------------------------------------------------------------------------------------------------------
RELEASE: 'Stage21-2 Training Recovery'
TAG    : stage21-2-training-recovery-v1
ID     : 371280600
DRAFT  : False
PRE    : True
ASSETS : 10
    stage21-2-recovery-epoch01.pt
      size    : 1.097 MiB (1,150,463 bytes)
      digest  : sha256:4df22b505830bf61573f4352c3d5e89480c8d0a74ad3887bf625fb2270044b24
      type    : application/octet-stream
      downloads: 0
    stage21-2-recovery-epoch02.pt
      size    : 1.097 MiB (1,150,783 bytes)

In [9]:
# =============================================================================
# STAGE26 — MONDAY RELEASE CORPUS STRUCTURE AUDIT
#
# PURPOSE:
#   Inspect the AUTHORITATIVE EXISTING Stage20 Monday compact corpus from
#   GitHub Releases.
#
# HARD RULE:
#   DO NOT RECREATE ANY CORPUS FROM THE PCAP.
#
# THIS CELL:
#   - reads the already-created GitHub release inventory;
#   - identifies the exact Monday release asset;
#   - downloads ONLY that existing release TAR + optional checksum sidecar;
#   - verifies exact GitHub release SHA256;
#   - lists all TAR members;
#   - reads ONLY small manifest/metadata/text members;
#   - DOES NOT extract bulk corpus payload;
#   - DOES NOT open the 10.08-GiB Monday PCAP;
#   - DOES NOT perform extraction;
#   - DOES NOT perform timing;
#   - DOES NOT access Thursday or Friday corpus contents;
#   - DOES NOT modify Git.
# =============================================================================

from __future__ import annotations

import os
import json
import hashlib
import shutil
import tarfile
import urllib.request
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. PATHS / FROZEN IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

EXTRACTION_ROOT = (
    STAGE26_ROOT
    / "extraction"
)

RELEASE_ROOT = (
    STAGE26_ROOT
    / "release_corpora"
)

MONDAY_ROOT = (
    RELEASE_ROOT
    / "Monday"
)

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

MONDAY_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


EXPECTED_HEAD = (
    "347d93f21d454cc5bda2c45889c890c67cdf0ecc"
)


# -----------------------------------------------------------------------------
# Existing release inventory generated in prior cell
# -----------------------------------------------------------------------------

RELEASE_INVENTORY = (
    EXTRACTION_ROOT
    / "stage26_github_release_corpus_inventory.json"
)


# -----------------------------------------------------------------------------
# Exact release identity from observed audit
# -----------------------------------------------------------------------------

EXPECTED_RELEASE_TAG = (
    "stage20-compact-corpora-v1"
)

EXPECTED_RELEASE_NAME = (
    "Stage20 Frozen Compact Corpora v1"
)

MONDAY_ASSET_NAME = (
    "stage20-Monday-compact-corpus-v1.tar"
)

MONDAY_SHA_ASSET_NAME = (
    "stage20-Monday-compact-corpus-v1.tar.sha256"
)

EXPECTED_MONDAY_TAR_SIZE = (
    595_261_440
)

EXPECTED_MONDAY_TAR_SHA256 = (
    "4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20"
)


MONDAY_TAR = (
    MONDAY_ROOT
    / MONDAY_ASSET_NAME
)

MONDAY_SHA_FILE = (
    MONDAY_ROOT
    / MONDAY_SHA_ASSET_NAME
)


OUTPUT_RECEIPT = (
    EXTRACTION_ROOT
    / "stage26_monday_release_corpus_structure_audit.json"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 118
    )

    print(text)

    print(
        "=" * 118
    )


def human_bytes(n):

    n = float(
        n
    )

    for unit in [
        "B",
        "KiB",
        "MiB",
        "GiB",
        "TiB",
    ]:

        if (
            n < 1024
            or
            unit == "TiB"
        ):

            return (
                f"{n:.3f} {unit}"
            )

        n /= 1024


def sha256_file(
    path,
    *,
    progress=False,
):

    path = Path(
        path
    )

    h = hashlib.sha256()

    total = 0

    next_report = (
        128 * 1024**2
    )


    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                16 * 1024 * 1024
            )


            if not chunk:
                break


            h.update(
                chunk
            )

            total += len(
                chunk
            )


            if (
                progress
                and
                total >= next_report
            ):

                print(
                    "  hashed",
                    human_bytes(
                        total
                    ),
                    flush=True,
                )

                next_report += (
                    128 * 1024**2
                )


    return h.hexdigest()


def git(*args):

    p = subprocess.run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )


    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )


    return p.stdout.strip()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )


    os.replace(
        tmp,
        path,
    )


def download_file(
    url,
    destination,
    headers,
):

    destination = Path(
        destination
    )

    partial = Path(
        str(destination)
        + ".part"
    )


    # Resume our own partial HTTP download if one exists.
    existing = (
        partial.stat().st_size
        if partial.exists()
        else 0
    )


    req_headers = dict(
        headers
    )


    if existing > 0:

        req_headers[
            "Range"
        ] = (
            f"bytes={existing}-"
        )


        print(
            "Resuming local release download from:",
            human_bytes(
                existing
            )
        )


    request = urllib.request.Request(
        url,
        headers=req_headers,
        method="GET",
    )


    with urllib.request.urlopen(
        request,
        timeout=300,
    ) as response:

        status_code = int(
            getattr(
                response,
                "status",
                200,
            )
        )


        if (
            existing > 0
            and
            status_code == 206
        ):

            mode = "ab"

        else:

            # Server did not honor Range or there was no partial.
            mode = "wb"

            existing = 0


        downloaded = existing

        next_report = (
            (
                downloaded
                //
                (
                    64 * 1024**2
                )
            )
            +
            1
        ) * (
            64 * 1024**2
        )


        with partial.open(
            mode
        ) as f:

            while True:

                chunk = response.read(
                    8 * 1024 * 1024
                )


                if not chunk:

                    break


                f.write(
                    chunk
                )

                downloaded += len(
                    chunk
                )


                if downloaded >= next_report:

                    print(
                        "  downloaded",
                        human_bytes(
                            downloaded
                        ),
                        flush=True,
                    )

                    next_report += (
                        64 * 1024**2
                    )


            f.flush()

            os.fsync(
                f.fileno()
            )


    os.replace(
        partial,
        destination,
    )


# =============================================================================
# 2. SCIENTIFIC GIT ANCHOR
# =============================================================================

banner(
    "MONDAY RELEASE CORPUS AUDIT :: SCIENTIFIC ANCHOR"
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD :",
    EXPECTED_HEAD
)

print(
    "Local HEAD    :",
    head
)

print(
    "origin/main   :",
    remote
)

print(
    "Repo clean    :",
    status == ""
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected local HEAD."
    )


if remote != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected origin/main."
    )


if status:

    raise RuntimeError(
        "Repository must remain clean."
    )


# =============================================================================
# 3. RELEASE INVENTORY GATE
# =============================================================================

banner(
    "MONDAY RELEASE CORPUS AUDIT :: RELEASE INVENTORY"
)


if not RELEASE_INVENTORY.exists():

    raise FileNotFoundError(
        RELEASE_INVENTORY
    )


inventory = json.loads(
    RELEASE_INVENTORY.read_text(
        encoding="utf-8"
    )
)


print(
    "Release count:",
    inventory[
        "release_count"
    ]
)

print(
    "Asset count:",
    inventory[
        "asset_count"
    ]
)


# Find exact Monday TAR + SHA sidecar from release metadata.
monday_asset = None
monday_sha_asset = None


for release in inventory[
    "releases"
]:

    if release[
        "tag_name"
    ] != EXPECTED_RELEASE_TAG:

        continue


    if release[
        "name"
    ] != EXPECTED_RELEASE_NAME:

        raise RuntimeError(
            "Unexpected release name for frozen corpus tag."
        )


    for asset in release[
        "assets"
    ]:

        if asset[
            "name"
        ] == MONDAY_ASSET_NAME:

            monday_asset = asset


        elif asset[
            "name"
        ] == MONDAY_SHA_ASSET_NAME:

            monday_sha_asset = asset


if monday_asset is None:

    raise RuntimeError(
        "Monday compact corpus release asset not found."
    )


if monday_sha_asset is None:

    raise RuntimeError(
        "Monday checksum sidecar release asset not found."
    )


print(
    "\nMonday TAR asset:"
)

print(
    "  name  :",
    monday_asset[
        "name"
    ]
)

print(
    "  size  :",
    monday_asset[
        "size_bytes"
    ]
)

print(
    "  digest:",
    monday_asset[
        "digest"
    ]
)


if int(
    monday_asset[
        "size_bytes"
    ]
) != EXPECTED_MONDAY_TAR_SIZE:

    raise RuntimeError(
        "Monday release asset size changed."
    )


expected_digest_field = (
    "sha256:"
    +
    EXPECTED_MONDAY_TAR_SHA256
)


if monday_asset[
    "digest"
] != expected_digest_field:

    raise RuntimeError(
        "Monday release asset digest changed."
    )


# =============================================================================
# 4. DISK SPACE
# =============================================================================

banner(
    "MONDAY RELEASE CORPUS AUDIT :: DISK SPACE"
)


usage = shutil.disk_usage(
    "/kaggle/working"
)


print(
    "Workspace free:",
    human_bytes(
        usage.free
    )
)

print(
    "Monday corpus TAR:",
    human_bytes(
        EXPECTED_MONDAY_TAR_SIZE
    )
)


# Require tar plus generous 512 MiB safety margin.
required = (
    EXPECTED_MONDAY_TAR_SIZE
    +
    512 * 1024**2
)


if (
    not MONDAY_TAR.exists()
    and
    usage.free < required
):

    raise RuntimeError(
        "Insufficient disk space for Monday Release corpus TAR."
    )


# =============================================================================
# 5. GITHUB AUTH — TOKEN NEVER PRINTED
# =============================================================================

banner(
    "MONDAY RELEASE CORPUS AUDIT :: GITHUB AUTH"
)


headers = {
    "Accept":
        "application/octet-stream",

    "User-Agent":
        "stage26-monday-release-corpus-audit",
}


try:

    from kaggle_secrets import UserSecretsClient


    token = UserSecretsClient().get_secret(
        "GITHUB_TOKEN"
    )


    if token:

        headers[
            "Authorization"
        ] = (
            f"Bearer {token}"
        )

        print(
            "[FOUND] GITHUB_TOKEN"
        )

    else:

        print(
            "[INFO] Public download mode."
        )


except Exception:

    token = None

    print(
        "[INFO] Public download mode."
    )


# =============================================================================
# 6. DOWNLOAD EXISTING RELEASE ASSETS ONLY
# =============================================================================

banner(
    "MONDAY RELEASE CORPUS AUDIT :: DOWNLOAD EXISTING RELEASE CORPUS"
)


if MONDAY_TAR.exists():

    print(
        "Monday TAR already present:"
    )

    print(
        " ",
        MONDAY_TAR
    )

else:

    print(
        "Downloading EXISTING GitHub Release corpus."
    )

    print(
        "No corpus reconstruction is occurring."
    )


    download_file(
        monday_asset[
            "browser_download_url"
        ],
        MONDAY_TAR,
        headers,
    )


if MONDAY_SHA_FILE.exists():

    print(
        "\nChecksum sidecar already present."
    )

else:

    download_file(
        monday_sha_asset[
            "browser_download_url"
        ],
        MONDAY_SHA_FILE,
        headers,
    )


if token is not None:

    del token


# =============================================================================
# 7. VERIFY RELEASE TAR EXACTLY
# =============================================================================

banner(
    "MONDAY RELEASE CORPUS AUDIT :: BYTE IDENTITY"
)


actual_size = int(
    MONDAY_TAR.stat().st_size
)


print(
    "Expected bytes:",
    EXPECTED_MONDAY_TAR_SIZE
)

print(
    "Actual bytes  :",
    actual_size
)


if actual_size != EXPECTED_MONDAY_TAR_SIZE:

    raise RuntimeError(
        "Monday Release corpus TAR size mismatch."
    )


print(
    "\nComputing Monday Release TAR SHA256..."
)


actual_sha = sha256_file(
    MONDAY_TAR,
    progress=True,
)


print(
    "\nExpected SHA256:"
)

print(
    " ",
    EXPECTED_MONDAY_TAR_SHA256
)

print(
    "Actual SHA256:"
)

print(
    " ",
    actual_sha
)


if actual_sha != EXPECTED_MONDAY_TAR_SHA256:

    raise RuntimeError(
        "Monday Release corpus TAR SHA256 mismatch."
    )


print(
    "\n[PASS] Existing Monday Release corpus is byte-identical."
)


# =============================================================================
# 8. VERIFY CHECKSUM SIDECAR CONTENT
# =============================================================================

banner(
    "MONDAY RELEASE CORPUS AUDIT :: RELEASE CHECKSUM SIDECAR"
)


sidecar_text = MONDAY_SHA_FILE.read_text(
    encoding="utf-8",
    errors="replace",
).strip()


print(
    sidecar_text
)


if EXPECTED_MONDAY_TAR_SHA256 not in sidecar_text:

    raise RuntimeError(
        "Release checksum sidecar does not contain expected TAR SHA256."
    )


print(
    "\n[PASS] Release checksum sidecar agrees."
)


# =============================================================================
# 9. INVENTORY TAR — NO BULK EXTRACTION
# =============================================================================

banner(
    "MONDAY RELEASE CORPUS AUDIT :: TAR MEMBER INVENTORY"
)


members = []


with tarfile.open(
    MONDAY_TAR,
    mode="r:",
) as tf:

    tar_members = tf.getmembers()


    print(
        "TAR member count:",
        len(
            tar_members
        )
    )


    total_member_bytes = 0


    for index, member in enumerate(
        tar_members,
        start=1,
    ):

        size = int(
            member.size
        )

        total_member_bytes += size


        record = {
            "index":
                index,

            "name":
                member.name,

            "size_bytes":
                size,

            "size_human":
                human_bytes(
                    size
                ),

            "is_file":
                member.isfile(),

            "is_dir":
                member.isdir(),

            "mode":
                oct(
                    member.mode
                ),
        }


        members.append(
            record
        )


        kind = (
            "FILE"
            if member.isfile()
            else
            "DIR "
            if member.isdir()
            else
            "OTHER"
        )


        print(
            f"[{index:03d}] "
            f"{kind:5s}  "
            f"{human_bytes(size):>12s}  "
            f"{member.name}"
        )


print(
    "\nTotal logical member bytes:",
    human_bytes(
        total_member_bytes
    )
)


# =============================================================================
# 10. CLASSIFY LIKELY CORPUS COMPONENTS
# =============================================================================

banner(
    "MONDAY RELEASE CORPUS AUDIT :: COMPONENT CLASSIFICATION"
)


metadata_keywords = [
    "manifest",
    "metadata",
    "meta",
    "provenance",
    "geometry",
    "schema",
    "index",
    "offset",
    "length",
    "sha256",
    "checksum",
    "readme",
    "receipt",
    "json",
    "txt",
    "csv",
]


payload_keywords = [
    "packet",
    "bytes",
    "blob",
    "data",
    "corpus",
    "payload",
    "flow",
    "image",
    "array",
    "npy",
    "npz",
    "parquet",
    "bin",
]


metadata_candidates = []

payload_candidates = []


for member in members:

    if not member[
        "is_file"
    ]:

        continue


    lower = member[
        "name"
    ].lower()


    meta_matches = [
        kw
        for kw in metadata_keywords
        if kw in lower
    ]


    payload_matches = [
        kw
        for kw in payload_keywords
        if kw in lower
    ]


    if meta_matches:

        metadata_candidates.append(
            {
                **member,
                "matched_keywords":
                    meta_matches,
            }
        )


    if payload_matches:

        payload_candidates.append(
            {
                **member,
                "matched_keywords":
                    payload_matches,
            }
        )


print(
    "Likely metadata/index members:",
    len(
        metadata_candidates
    )
)


for item in metadata_candidates:

    print(
        "  META   ",
        item[
            "name"
        ],
        "|",
        item[
            "size_human"
        ]
    )


print(
    "\nLikely corpus/payload members:",
    len(
        payload_candidates
    )
)


for item in payload_candidates:

    print(
        "  PAYLOAD",
        item[
            "name"
        ],
        "|",
        item[
            "size_human"
        ]
    )


# =============================================================================
# 11. READ SMALL METADATA MEMBERS IN MEMORY ONLY
# =============================================================================

banner(
    "MONDAY RELEASE CORPUS AUDIT :: SMALL METADATA CONTENT"
)


# We deliberately do not extract files to disk.
# Only metadata-like members <= 2 MiB are read into memory.
small_metadata = []


with tarfile.open(
    MONDAY_TAR,
    mode="r:",
) as tf:

    member_lookup = {
        member.name:
            member
        for member in tf.getmembers()
    }


    for candidate in metadata_candidates:

        if candidate[
            "size_bytes"
        ] > (
            2 * 1024**2
        ):

            continue


        member = member_lookup[
            candidate[
                "name"
            ]
        ]


        file_obj = tf.extractfile(
            member
        )


        if file_obj is None:

            continue


        raw = file_obj.read()


        digest = hashlib.sha256(
            raw
        ).hexdigest()


        text = raw.decode(
            "utf-8",
            errors="replace",
        )


        record = {
            "name":
                member.name,

            "size_bytes":
                len(
                    raw
                ),

            "sha256":
                digest,

            "text_preview":
                text[
                    :20000
                ],
        }


        small_metadata.append(
            record
        )


        print(
            "\n"
            + "-" * 118
        )

        print(
            member.name
        )

        print(
            "size  :",
            len(
                raw
            )
        )

        print(
            "SHA256:",
            digest
        )

        print(
            "CONTENT/PREVIEW:"
        )

        print(
            text[
                :20000
            ]
        )


# =============================================================================
# 12. IDENTIFY REPRESENTATION BOUNDARY
# =============================================================================

banner(
    "MONDAY RELEASE CORPUS AUDIT :: STAGE26 ROLE MAP"
)


member_names = [
    x[
        "name"
    ]
    for x in members
]


print(
    "Authoritative existing downstream source:"
)

print(
    "  GitHub Release tag:",
    EXPECTED_RELEASE_TAG
)

print(
    "  Monday asset     :",
    MONDAY_ASSET_NAME
)

print(
    "  TAR SHA256       :",
    actual_sha
)


print(
    "\nRaw PCAP role:"
)

print(
    "  RAW-CAPTURE EXTRACTION BENCHMARK ONLY"
)


print(
    "\nRelease corpus role:"
)

print(
    "  AUTHORITATIVE EXISTING STAGE20 DOWNSTREAM CORPUS"
)


print(
    "\nExplicitly forbidden:"
)

print(
    "  Rebuilding/recreating this corpus from Monday PCAP"
)


# =============================================================================
# 13. WRITE LOCAL STRUCTURE AUDIT
# =============================================================================

receipt = {
    "schema":
        "stage26_monday_release_corpus_structure_audit_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "scientific_parent":
        EXPECTED_HEAD,

    "release": {
        "tag":
            EXPECTED_RELEASE_TAG,

        "name":
            EXPECTED_RELEASE_NAME,

        "asset":
            MONDAY_ASSET_NAME,

        "size_bytes":
            actual_size,

        "sha256":
            actual_sha,

        "checksum_sidecar":
            MONDAY_SHA_ASSET_NAME,

        "checksum_sidecar_text":
            sidecar_text,
    },

    "tar": {
        "member_count":
            len(
                members
            ),

        "members":
            members,

        "logical_member_bytes":
            total_member_bytes,

        "metadata_candidates":
            metadata_candidates,

        "payload_candidates":
            payload_candidates,

        "small_metadata":
            small_metadata,
    },

    "scientific_role": {
        "existing_release_corpus_is_authoritative":
            True,

        "recreate_corpus_from_pcap":
            False,

        "pcap_role":
            "RAW_EXTRACTION_BENCHMARK_ONLY",

        "release_corpus_role":
            (
                "AUTHORITATIVE_DOWNSTREAM_FLOW_PACKET_"
                "REPRESENTATION_SOURCE"
            ),

        "bulk_corpus_extracted":
            False,

        "pcap_opened":
            False,

        "pcap_packets_iterated":
            False,

        "extraction_performed":
            False,

        "timing_performed":
            False,

        "Thursday_corpus_content_accessed":
            False,

        "Friday_corpus_content_accessed":
            False,

        "models_loaded":
            False,

        "inference":
            False,

        "gpu":
            False,

        "git_modified":
            False,
    },
}


atomic_json(
    OUTPUT_RECEIPT,
    receipt,
)


receipt_sha = sha256_file(
    OUTPUT_RECEIPT
)


# =============================================================================
# 14. FINAL GIT AUDIT
# =============================================================================

banner(
    "MONDAY RELEASE CORPUS AUDIT :: FINAL AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "Local HEAD :",
    final_head
)

print(
    "Remote HEAD:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_HEAD:

    raise RuntimeError(
        "HEAD changed."
    )


if final_remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed."
    )


if final_status:

    raise RuntimeError(
        "Repository changed."
    )


print(
    "\nStructure audit:"
)

print(
    " ",
    OUTPUT_RECEIPT
)

print(
    " SHA256:",
    receipt_sha
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  EXISTING MONDAY RELEASE CORPUS VERIFIED : YES"
)

print(
    "  RELEASE TAR FULLY EXTRACTED             : NO"
)

print(
    "  CORPUS RECREATED FROM PCAP              : NO"
)

print(
    "  PCAP OPENED                             : NO"
)

print(
    "  EXTRACTION                              : NO"
)

print(
    "  EXTRACTION TIMING                       : NO"
)

print(
    "  THURSDAY/FRIDAY CORPUS CONTENT          : NOT ACCESSED"
)

print(
    "  GPU                                     : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Map exact TAR members to Stage26 preprocessing boundaries,"
)

print(
    "  then freeze Stage26-4 using PCAP only for raw-extraction cost"
)

print(
    "  and this Release corpus for all existing downstream data."
)


MONDAY RELEASE CORPUS AUDIT :: SCIENTIFIC ANCHOR
Expected HEAD : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Local HEAD    : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
origin/main   : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Repo clean    : True

MONDAY RELEASE CORPUS AUDIT :: RELEASE INVENTORY
Release count: 2
Asset count: 20

Monday TAR asset:
  name  : stage20-Monday-compact-corpus-v1.tar
  size  : 595261440
  digest: sha256:4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20

MONDAY RELEASE CORPUS AUDIT :: DISK SPACE
Workspace free: 7.141 GiB
Monday corpus TAR: 567.686 MiB

MONDAY RELEASE CORPUS AUDIT :: GITHUB AUTH
[FOUND] GITHUB_TOKEN

MONDAY RELEASE CORPUS AUDIT :: DOWNLOAD EXISTING RELEASE CORPUS
No corpus reconstruction is occurring.
  downloaded 64.000 MiB
  downloaded 128.000 MiB
  downloaded 192.000 MiB
  downloaded 256.000 MiB
  downloaded 320.000 MiB
  downloaded 384.000 MiB
  downloaded 448.000 MiB
  downloaded 512.000 MiB

MONDAY RELEASE CORPUS AUDIT :: BYTE ID

In [10]:
# =============================================================================
# STAGE26 — MONDAY RELEASE CORPUS INTERNAL GEOMETRY AUDIT
#
# PURPOSE:
#   Establish the exact scientific boundary and internal geometry of the
#   already-frozen Stage20 Monday compact corpus.
#
# HARD RULE:
#   NOTHING is recreated from the PCAP.
#
# THIS CELL:
#   - verifies the four expected TAR members;
#   - computes SHA256 for each member directly from the TAR;
#   - loads only the three .npy arrays from the existing Release TAR;
#   - reports dtype / shape / min / max / monotonicity;
#   - checks structural relationships;
#   - DOES NOT extract encoded_bytes.bin to disk;
#   - DOES NOT open the Monday PCAP;
#   - DOES NOT perform extraction;
#   - DOES NOT perform timing;
#   - DOES NOT load models;
#   - DOES NOT use GPU;
#   - DOES NOT modify Git.
# =============================================================================

from __future__ import annotations

import io
import os
import json
import hashlib
import tarfile
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np


# =============================================================================
# 0. PATHS / FROZEN IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

EXTRACTION_ROOT = (
    STAGE26_ROOT
    / "extraction"
)

MONDAY_ROOT = (
    STAGE26_ROOT
    / "release_corpora"
    / "Monday"
)

MONDAY_TAR = (
    MONDAY_ROOT
    / "stage20-Monday-compact-corpus-v1.tar"
)

EXPECTED_HEAD = (
    "347d93f21d454cc5bda2c45889c890c67cdf0ecc"
)

EXPECTED_TAR_SIZE = (
    595_261_440
)

EXPECTED_TAR_SHA256 = (
    "4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20"
)


# -----------------------------------------------------------------------------
# Expected historical member identities
# -----------------------------------------------------------------------------

EXPECTED_MEMBERS = {
    "Monday/encoded_bytes.bin": {
        "sha256":
            "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",
    },

    "Monday/flow_offsets.npy": {
        "sha256":
            "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",
    },

    "Monday/labels.npy": {
        "sha256":
            "48792b8d6a127b35342cb0789baa6c54396f1100a60ce7225daf08d1c3530424",
    },

    "Monday/packet_lengths.npy": {
        "sha256":
            "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
    },
}


STRUCTURE_AUDIT = (
    EXTRACTION_ROOT
    / "stage26_monday_release_corpus_structure_audit.json"
)

EXPECTED_STRUCTURE_AUDIT_SHA256 = (
    "a0ed17fa8d26049f53766e32981dff72724d79fbd573e7aab79c926a36833c3b"
)


OUTPUT = (
    EXTRACTION_ROOT
    / "stage26_monday_release_corpus_geometry_audit.json"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 118
    )

    print(text)

    print(
        "=" * 118
    )


def git(*args):

    p = subprocess.run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def sha256_stream(file_obj):

    h = hashlib.sha256()

    total = 0

    while True:

        chunk = file_obj.read(
            8 * 1024 * 1024
        )

        if not chunk:
            break

        h.update(
            chunk
        )

        total += len(
            chunk
        )

    return (
        h.hexdigest(),
        total,
    )


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def array_summary(
    name,
    arr,
):

    summary = {
        "name":
            name,

        "dtype":
            str(
                arr.dtype
            ),

        "shape":
            list(
                arr.shape
            ),

        "ndim":
            int(
                arr.ndim
            ),

        "size":
            int(
                arr.size
            ),

        "nbytes":
            int(
                arr.nbytes
            ),
    }


    if arr.size > 0:

        summary[
            "min"
        ] = (
            arr.min().item()
        )

        summary[
            "max"
        ] = (
            arr.max().item()
        )

        summary[
            "first_10"
        ] = (
            arr.reshape(
                -1
            )[
                :10
            ].tolist()
        )

        summary[
            "last_10"
        ] = (
            arr.reshape(
                -1
            )[
                -10:
            ].tolist()
        )


    return summary


# =============================================================================
# 2. SCIENTIFIC GIT ANCHOR
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY :: SCIENTIFIC ANCHOR"
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD :",
    EXPECTED_HEAD
)

print(
    "Local HEAD    :",
    head
)

print(
    "origin/main   :",
    remote
)

print(
    "Repo clean    :",
    status == ""
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected local HEAD."
    )


if remote != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected origin/main."
    )


if status:

    raise RuntimeError(
        "Repository must remain clean."
    )


# =============================================================================
# 3. UPSTREAM AUDIT GATE
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY :: UPSTREAM AUDIT GATE"
)


if not STRUCTURE_AUDIT.exists():

    raise FileNotFoundError(
        STRUCTURE_AUDIT
    )


structure_sha = sha256_file(
    STRUCTURE_AUDIT
)


print(
    "Structure audit SHA256:"
)

print(
    " ",
    structure_sha
)


if (
    structure_sha
    !=
    EXPECTED_STRUCTURE_AUDIT_SHA256
):

    raise RuntimeError(
        "Monday Release structure audit changed."
    )


# =============================================================================
# 4. TAR BYTE IDENTITY GATE
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY :: TAR IDENTITY"
)


if not MONDAY_TAR.exists():

    raise FileNotFoundError(
        MONDAY_TAR
    )


actual_tar_size = int(
    MONDAY_TAR.stat().st_size
)


print(
    "Expected TAR bytes:",
    EXPECTED_TAR_SIZE
)

print(
    "Actual TAR bytes  :",
    actual_tar_size
)


if actual_tar_size != EXPECTED_TAR_SIZE:

    raise RuntimeError(
        "Monday Release TAR size mismatch."
    )


actual_tar_sha = sha256_file(
    MONDAY_TAR
)


print(
    "Expected TAR SHA256:"
)

print(
    " ",
    EXPECTED_TAR_SHA256
)

print(
    "Actual TAR SHA256:"
)

print(
    " ",
    actual_tar_sha
)


if actual_tar_sha != EXPECTED_TAR_SHA256:

    raise RuntimeError(
        "Monday Release TAR SHA256 mismatch."
    )


print(
    "\n[PASS] Exact frozen Release TAR."
)


# =============================================================================
# 5. MEMBER IDENTITY AUDIT
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY :: MEMBER HASH AUDIT"
)


member_records = {}

loaded_arrays = {}


with tarfile.open(
    MONDAY_TAR,
    mode="r:",
) as tf:

    actual_names = {
        m.name
        for m in tf.getmembers()
        if m.isfile()
    }


    expected_names = set(
        EXPECTED_MEMBERS
    )


    print(
        "Expected member count:",
        len(
            expected_names
        )
    )

    print(
        "Actual member count  :",
        len(
            actual_names
        )
    )


    if actual_names != expected_names:

        print(
            "\nExpected:"
        )

        for x in sorted(
            expected_names
        ):

            print(
                " ",
                x
            )


        print(
            "\nActual:"
        )

        for x in sorted(
            actual_names
        ):

            print(
                " ",
                x
            )


        raise RuntimeError(
            "Unexpected Monday corpus member set."
        )


    for member_name in sorted(
        expected_names
    ):

        member = tf.getmember(
            member_name
        )

        file_obj = tf.extractfile(
            member
        )


        if file_obj is None:

            raise RuntimeError(
                f"Could not read TAR member: {member_name}"
            )


        digest, byte_count = sha256_stream(
            file_obj
        )


        expected_digest = (
            EXPECTED_MEMBERS[
                member_name
            ][
                "sha256"
            ]
        )


        passed = (
            digest
            ==
            expected_digest
        )


        print(
            f"{'PASS' if passed else 'FAIL'} "
            f"{digest}  "
            f"{member_name}  "
            f"({byte_count:,} bytes)"
        )


        if not passed:

            raise RuntimeError(
                f"Member SHA mismatch: {member_name}"
            )


        if byte_count != int(
            member.size
        ):

            raise RuntimeError(
                f"Member byte-count mismatch: {member_name}"
            )


        member_records[
            member_name
        ] = {
            "size_bytes":
                int(
                    member.size
                ),

            "sha256":
                digest,
        }


# =============================================================================
# 6. LOAD ONLY EXISTING .NPY ARRAYS FROM TAR
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY :: LOAD EXISTING INDEX ARRAYS"
)


with tarfile.open(
    MONDAY_TAR,
    mode="r:",
) as tf:

    for member_name in [
        "Monday/flow_offsets.npy",
        "Monday/labels.npy",
        "Monday/packet_lengths.npy",
    ]:

        member = tf.getmember(
            member_name
        )

        file_obj = tf.extractfile(
            member
        )


        if file_obj is None:

            raise RuntimeError(
                f"Cannot read {member_name}"
            )


        raw = file_obj.read()


        arr = np.load(
            io.BytesIO(
                raw
            ),
            allow_pickle=False,
        )


        loaded_arrays[
            member_name
        ] = arr


        summary = array_summary(
            member_name,
            arr,
        )


        print(
            "\n",
            member_name,
        )

        print(
            "  dtype :",
            summary[
                "dtype"
            ]
        )

        print(
            "  shape :",
            summary[
                "shape"
            ]
        )

        print(
            "  size  :",
            f"{summary['size']:,}"
        )

        print(
            "  nbytes:",
            f"{summary['nbytes']:,}"
        )

        if arr.size:

            print(
                "  min   :",
                summary[
                    "min"
                ]
            )

            print(
                "  max   :",
                summary[
                    "max"
                ]
            )

            print(
                "  first :",
                summary[
                    "first_10"
                ]
            )

            print(
                "  last  :",
                summary[
                    "last_10"
                ]
            )


# =============================================================================
# 7. STRUCTURAL RELATIONSHIP AUDIT
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY :: STRUCTURAL RELATIONSHIPS"
)


flow_offsets = loaded_arrays[
    "Monday/flow_offsets.npy"
]

labels = loaded_arrays[
    "Monday/labels.npy"
]

packet_lengths = loaded_arrays[
    "Monday/packet_lengths.npy"
]


if flow_offsets.ndim != 1:

    raise RuntimeError(
        "flow_offsets.npy is not 1-D."
    )


if labels.ndim != 1:

    raise RuntimeError(
        "labels.npy is not 1-D."
    )


if packet_lengths.ndim != 1:

    raise RuntimeError(
        "packet_lengths.npy is not 1-D."
    )


offset_monotonic = bool(
    np.all(
        flow_offsets[
            1:
        ]
        >=
        flow_offsets[
            :-1
        ]
    )
)


offset_strict = bool(
    np.all(
        flow_offsets[
            1:
        ]
        >
        flow_offsets[
            :-1
        ]
    )
)


packet_lengths_nonnegative = bool(
    np.all(
        packet_lengths
        >=
        0
    )
)


packet_lengths_positive = bool(
    np.all(
        packet_lengths
        >
        0
    )
)


label_values, label_counts = np.unique(
    labels,
    return_counts=True,
)


encoded_bytes_size = (
    member_records[
        "Monday/encoded_bytes.bin"
    ][
        "size_bytes"
    ]
)


packet_length_sum = int(
    packet_lengths.astype(
        np.uint64,
        copy=False,
    ).sum(
        dtype=np.uint64
    )
)


print(
    "flow_offsets entries      :",
    f"{flow_offsets.size:,}"
)

print(
    "labels entries            :",
    f"{labels.size:,}"
)

print(
    "packet_lengths entries    :",
    f"{packet_lengths.size:,}"
)

print(
    "encoded_bytes.bin bytes   :",
    f"{encoded_bytes_size:,}"
)

print(
    "sum(packet_lengths)       :",
    f"{packet_length_sum:,}"
)

print(
    "offsets nondecreasing     :",
    offset_monotonic
)

print(
    "offsets strictly increase :",
    offset_strict
)

print(
    "packet lengths >= 0       :",
    packet_lengths_nonnegative
)

print(
    "packet lengths > 0        :",
    packet_lengths_positive
)


print(
    "\nLabels:"
)

for value, count in zip(
    label_values.tolist(),
    label_counts.tolist(),
):

    print(
        f"  {value!r}: {count:,}"
    )


# =============================================================================
# 8. TEST POSSIBLE OFFSET INTERPRETATIONS
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY :: OFFSET SEMANTICS DIAGNOSTIC"
)


relationships = {}


relationships[
    "flow_offsets_size_equals_labels_size"
] = bool(
    flow_offsets.size
    ==
    labels.size
)


relationships[
    "flow_offsets_size_equals_labels_plus_one"
] = bool(
    flow_offsets.size
    ==
    labels.size
    +
    1
)


relationships[
    "last_flow_offset_equals_packet_count"
] = (
    bool(
        int(
            flow_offsets[
                -1
            ]
        )
        ==
        packet_lengths.size
    )
    if flow_offsets.size
    else False
)


relationships[
    "last_flow_offset_plus_one_equals_packet_count"
] = (
    bool(
        int(
            flow_offsets[
                -1
            ]
        )
        +
        1
        ==
        packet_lengths.size
    )
    if flow_offsets.size
    else False
)


relationships[
    "sum_packet_lengths_equals_encoded_bytes"
] = bool(
    packet_length_sum
    ==
    encoded_bytes_size
)


relationships[
    "first_flow_offset"
] = (
    int(
        flow_offsets[
            0
        ]
    )
    if flow_offsets.size
    else None
)


relationships[
    "last_flow_offset"
] = (
    int(
        flow_offsets[
            -1
        ]
    )
    if flow_offsets.size
    else None
)


for key, value in relationships.items():

    print(
        f"{key:52s}: {value}"
    )


# =============================================================================
# 9. DERIVE FLOW PACKET COUNTS ONLY IF OFFSETS SUPPORT IT
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY :: FLOW GEOMETRY"
)


flow_packet_counts = None
offset_semantics = "UNRESOLVED"


if (
    flow_offsets.size
    ==
    labels.size
    +
    1
    and
    int(
        flow_offsets[
            0
        ]
    )
    == 0
    and
    int(
        flow_offsets[
            -1
        ]
    )
    ==
    packet_lengths.size
):

    flow_packet_counts = np.diff(
        flow_offsets.astype(
            np.int64,
            copy=False,
        )
    )

    offset_semantics = (
        "BOUNDARY_OFFSETS_LENGTH_N_FLOWS_PLUS_ONE"
    )


elif (
    flow_offsets.size
    ==
    labels.size
    and
    int(
        flow_offsets[
            0
        ]
    )
    == 0
):

    # Candidate interpretation:
    # each value is the first-packet index for a flow; final boundary is
    # packet_lengths.size.
    boundaries = np.empty(
        flow_offsets.size
        +
        1,
        dtype=np.int64,
    )

    boundaries[
        :-1
    ] = flow_offsets.astype(
        np.int64,
        copy=False,
    )

    boundaries[
        -1
    ] = packet_lengths.size


    if np.all(
        boundaries[
            1:
        ]
        >=
        boundaries[
            :-1
        ]
    ):

        flow_packet_counts = np.diff(
            boundaries
        )

        offset_semantics = (
            "FLOW_START_PACKET_INDEX_LENGTH_N_FLOWS"
        )


print(
    "Resolved offset interpretation:"
)

print(
    " ",
    offset_semantics
)


flow_geometry = {
    "offset_semantics":
        offset_semantics,
}


if flow_packet_counts is not None:

    print(
        "Flow count derived         :",
        f"{flow_packet_counts.size:,}"
    )

    print(
        "Min packets/flow           :",
        int(
            flow_packet_counts.min()
        )
    )

    print(
        "Max packets/flow           :",
        int(
            flow_packet_counts.max()
        )
    )

    print(
        "Median packets/flow        :",
        float(
            np.median(
                flow_packet_counts
            )
        )
    )

    print(
        "Mean packets/flow          :",
        float(
            flow_packet_counts.mean()
        )
    )

    print(
        "Zero-packet flows          :",
        int(
            np.count_nonzero(
                flow_packet_counts
                ==
                0
            )
        )
    )

    print(
        "Single-packet flows        :",
        int(
            np.count_nonzero(
                flow_packet_counts
                ==
                1
            )
        )
    )


    flow_geometry.update(
        {
            "flow_count":
                int(
                    flow_packet_counts.size
                ),

            "min_packets_per_flow":
                int(
                    flow_packet_counts.min()
                ),

            "max_packets_per_flow":
                int(
                    flow_packet_counts.max()
                ),

            "median_packets_per_flow":
                float(
                    np.median(
                        flow_packet_counts
                    )
                ),

            "mean_packets_per_flow":
                float(
                    flow_packet_counts.mean()
                ),

            "zero_packet_flows":
                int(
                    np.count_nonzero(
                        flow_packet_counts
                        ==
                        0
                    )
                ),

            "single_packet_flows":
                int(
                    np.count_nonzero(
                        flow_packet_counts
                        ==
                        1
                    )
                ),
        }
    )


else:

    print(
        "Flow packet counts cannot be safely derived"
    )

    print(
        "from array geometry alone; no semantics invented."
    )


# =============================================================================
# 10. ARRAY SUMMARIES
# =============================================================================

array_summaries = {}


for name, arr in loaded_arrays.items():

    array_summaries[
        name
    ] = array_summary(
        name,
        arr,
    )


# =============================================================================
# 11. WRITE AUDIT RECEIPT
# =============================================================================

receipt = {
    "schema":
        "stage26_monday_release_corpus_geometry_audit_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "scientific_parent":
        EXPECTED_HEAD,

    "release_corpus": {
        "tag":
            "stage20-compact-corpora-v1",

        "asset":
            "stage20-Monday-compact-corpus-v1.tar",

        "tar_size_bytes":
            actual_tar_size,

        "tar_sha256":
            actual_tar_sha,

        "member_records":
            member_records,
    },

    "arrays":
        array_summaries,

    "relationships": {
        **relationships,

        "flow_offsets_nondecreasing":
            offset_monotonic,

        "flow_offsets_strictly_increasing":
            offset_strict,

        "packet_lengths_nonnegative":
            packet_lengths_nonnegative,

        "packet_lengths_positive":
            packet_lengths_positive,

        "packet_length_sum":
            packet_length_sum,

        "encoded_bytes_size":
            encoded_bytes_size,

        "labels": {
            str(
                value
            ):
                int(
                    count
                )
            for value, count in zip(
                label_values.tolist(),
                label_counts.tolist(),
            )
        },
    },

    "flow_geometry":
        flow_geometry,

    "scientific_boundary": {
        "authoritative_existing_corpus":
            True,

        "corpus_recreated_from_pcap":
            False,

        "encoded_bytes_extracted_to_disk":
            False,

        "npy_arrays_read_from_existing_release_tar":
            True,

        "pcap_opened":
            False,

        "pcap_packets_iterated":
            False,

        "raw_extraction_performed":
            False,

        "timing_performed":
            False,

        "models_loaded":
            False,

        "labels_used_for_model_selection":
            False,

        "Thursday_accessed":
            False,

        "Friday_accessed":
            False,

        "gpu_used":
            False,

        "git_modified":
            False,
    },

    "stage26_role": {
        "pcap":
            "RAW_CAPTURE_TO_FLOW_RECONSTRUCTION_BENCHMARK_ONLY",

        "release_corpus":
            (
                "AUTHORITATIVE_EXISTING_DOWNSTREAM_PACKET_FLOW_CORPUS_"
                "FOR_REPRESENTATION_PREPROCESSING_MEASUREMENT"
            ),

        "corpus_regeneration":
            "FORBIDDEN",
    },
}


atomic_json(
    OUTPUT,
    receipt,
)


receipt_sha = sha256_file(
    OUTPUT
)


# =============================================================================
# 12. FINAL AUDIT
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY :: FINAL AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "Local HEAD :",
    final_head
)

print(
    "Remote HEAD:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_HEAD:

    raise RuntimeError(
        "HEAD changed."
    )


if final_remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed."
    )


if final_status:

    raise RuntimeError(
        "Repository changed."
    )


banner(
    "MONDAY RELEASE CORPUS GEOMETRY AUDIT COMPLETE"
)


print(
    "AUTHORITATIVE RELEASE CORPUS:"
)

print(
    "  stage20-compact-corpora-v1"
)

print(
    "  stage20-Monday-compact-corpus-v1.tar"
)


print(
    "\nMEMBER HASHES:"
)

for name in sorted(
    member_records
):

    print(
        " ",
        member_records[
            name
        ][
            "sha256"
        ],
        name
    )


print(
    "\nARRAY GEOMETRY:"
)

for name, summary in array_summaries.items():

    print(
        f"  {name}:"
    )

    print(
        f"    dtype={summary['dtype']} "
        f"shape={summary['shape']} "
        f"size={summary['size']:,}"
    )


print(
    "\nOFFSET SEMANTICS:"
)

print(
    " ",
    offset_semantics
)


print(
    "\nGEOMETRY AUDIT:"
)

print(
    " ",
    OUTPUT
)

print(
    " SHA256:",
    receipt_sha
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  EXISTING RELEASE CORPUS VERIFIED : YES"
)

print(
    "  MEMBER BYTE IDENTITIES VERIFIED  : YES"
)

print(
    "  CORPUS RECREATED FROM PCAP       : NO"
)

print(
    "  PCAP OPENED                      : NO"
)

print(
    "  RAW EXTRACTION                   : NO"
)

print(
    "  TIMING                           : NO"
)

print(
    "  MODEL INFERENCE                  : NO"
)

print(
    "  GPU                              : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Use these exact corpus semantics to freeze Stage26-4:"
)

print(
    "    PCAP -> flow reconstruction cost      [raw benchmark only]"
)

print(
    "    Release corpus -> representation cost [authoritative corpus]"
)

print(
    "    representation -> inference           [already Stage26-2]"
)


MONDAY CORPUS GEOMETRY :: SCIENTIFIC ANCHOR
Expected HEAD : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Local HEAD    : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
origin/main   : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Repo clean    : True

MONDAY CORPUS GEOMETRY :: UPSTREAM AUDIT GATE
Structure audit SHA256:
  a0ed17fa8d26049f53766e32981dff72724d79fbd573e7aab79c926a36833c3b

MONDAY CORPUS GEOMETRY :: TAR IDENTITY
Expected TAR bytes: 595261440
Actual TAR bytes  : 595261440
Expected TAR SHA256:
  4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20
Actual TAR SHA256:
  4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20

[PASS] Exact frozen Release TAR.

MONDAY CORPUS GEOMETRY :: MEMBER HASH AUDIT
Expected member count: 4
Actual member count  : 4
PASS 27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c  Monday/encoded_bytes.bin  (522,845,159 bytes)
PASS 3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e  Monday/flow_offsets.npy  (4,228,20

RuntimeError: packet_lengths.npy is not 1-D.

In [11]:
# =============================================================================
# STAGE26 — MONDAY RELEASE CORPUS GEOMETRY RECOVERY
#
# Narrow correction after discovering:
#
#   packet_lengths.npy shape = (528509, 64)
#
# NO:
#   - PCAP access
#   - corpus regeneration
#   - extraction timing
#   - model loading
#   - GPU
#   - Git modification
#
# PURPOSE:
#   Resolve the exact semantics linking:
#
#       encoded_bytes.bin
#       flow_offsets.npy
#       packet_lengths.npy
#       labels.npy
#
#   using ONLY the existing frozen GitHub Release corpus.
# =============================================================================

from __future__ import annotations

import io
import os
import json
import hashlib
import tarfile
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np


# =============================================================================
# 0. FROZEN PATHS / IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

EXTRACTION_ROOT = (
    STAGE26_ROOT
    / "extraction"
)

MONDAY_TAR = (
    STAGE26_ROOT
    / "release_corpora"
    / "Monday"
    / "stage20-Monday-compact-corpus-v1.tar"
)

EXPECTED_HEAD = (
    "347d93f21d454cc5bda2c45889c890c67cdf0ecc"
)

EXPECTED_TAR_SIZE = (
    595_261_440
)

EXPECTED_TAR_SHA256 = (
    "4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20"
)

EXPECTED_FLOW_COUNT = (
    528_509
)

EXPECTED_ENCODED_BYTES = (
    522_845_159
)


EXPECTED_MEMBER_SHA256 = {
    "Monday/encoded_bytes.bin":
        "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",

    "Monday/flow_offsets.npy":
        "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",

    "Monday/labels.npy":
        "48792b8d6a127b35342cb0789baa6c54396f1100a60ce7225daf08d1c3530424",

    "Monday/packet_lengths.npy":
        "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
}


STRUCTURE_AUDIT = (
    EXTRACTION_ROOT
    / "stage26_monday_release_corpus_structure_audit.json"
)

EXPECTED_STRUCTURE_AUDIT_SHA256 = (
    "a0ed17fa8d26049f53766e32981dff72724d79fbd573e7aab79c926a36833c3b"
)


OUTPUT = (
    EXTRACTION_ROOT
    / "stage26_monday_release_corpus_geometry_audit.json"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 118
    )

    print(text)

    print(
        "=" * 118
    )


def git(*args):

    p = subprocess.run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


# =============================================================================
# 2. SCIENTIFIC ANCHOR
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY RECOVERY :: SCIENTIFIC ANCHOR"
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD :",
    EXPECTED_HEAD
)

print(
    "Local HEAD    :",
    head
)

print(
    "origin/main   :",
    remote
)

print(
    "Repo clean    :",
    status == ""
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected local HEAD."
    )


if remote != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected origin/main."
    )


if status:

    raise RuntimeError(
        "Repository must remain clean."
    )


# =============================================================================
# 3. UPSTREAM STRUCTURE AUDIT GATE
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY RECOVERY :: UPSTREAM AUDIT"
)


actual_structure_sha = sha256_file(
    STRUCTURE_AUDIT
)


print(
    "Expected structure audit SHA256:"
)

print(
    " ",
    EXPECTED_STRUCTURE_AUDIT_SHA256
)

print(
    "Actual structure audit SHA256:"
)

print(
    " ",
    actual_structure_sha
)


if (
    actual_structure_sha
    !=
    EXPECTED_STRUCTURE_AUDIT_SHA256
):

    raise RuntimeError(
        "Upstream Monday corpus structure audit changed."
    )


# =============================================================================
# 4. TAR PRESENCE GATE
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY RECOVERY :: TAR GATE"
)


if not MONDAY_TAR.exists():

    raise FileNotFoundError(
        MONDAY_TAR
    )


tar_size = int(
    MONDAY_TAR.stat().st_size
)


print(
    "Expected TAR size:",
    EXPECTED_TAR_SIZE
)

print(
    "Actual TAR size  :",
    tar_size
)


if tar_size != EXPECTED_TAR_SIZE:

    raise RuntimeError(
        "Frozen Monday TAR size changed."
    )


# We deliberately do NOT repeat the full 568 MiB TAR hash here.
# The immediately preceding successful structure audit already verified it,
# and that audit is hash-gated above.
print(
    "Full TAR SHA256 previously verified by upstream audit:"
)

print(
    " ",
    EXPECTED_TAR_SHA256
)


# =============================================================================
# 5. LOAD EXISTING ARRAYS DIRECTLY FROM RELEASE TAR
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY RECOVERY :: LOAD ARRAYS"
)


arrays = {}


with tarfile.open(
    MONDAY_TAR,
    mode="r:",
) as tf:

    for name in [
        "Monday/flow_offsets.npy",
        "Monday/labels.npy",
        "Monday/packet_lengths.npy",
    ]:

        member = tf.getmember(
            name
        )

        f = tf.extractfile(
            member
        )

        if f is None:

            raise RuntimeError(
                f"Unable to read {name}"
            )


        raw = f.read()


        digest = hashlib.sha256(
            raw
        ).hexdigest()


        if (
            digest
            !=
            EXPECTED_MEMBER_SHA256[
                name
            ]
        ):

            raise RuntimeError(
                f"Member SHA mismatch: {name}"
            )


        arr = np.load(
            io.BytesIO(
                raw
            ),
            allow_pickle=False,
        )


        arrays[
            name
        ] = arr


        print(
            name
        )

        print(
            "  SHA256:",
            digest
        )

        print(
            "  dtype :",
            arr.dtype
        )

        print(
            "  shape :",
            arr.shape
        )

        print(
            "  size  :",
            f"{arr.size:,}"
        )


# =============================================================================
# 6. CORRECT SHAPE GATES
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY RECOVERY :: SHAPE GATES"
)


flow_offsets = arrays[
    "Monday/flow_offsets.npy"
]

labels = arrays[
    "Monday/labels.npy"
]

packet_lengths = arrays[
    "Monday/packet_lengths.npy"
]


print(
    "flow_offsets shape :",
    flow_offsets.shape
)

print(
    "labels shape       :",
    labels.shape
)

print(
    "packet_lengths     :",
    packet_lengths.shape
)


if flow_offsets.ndim != 1:

    raise RuntimeError(
        "flow_offsets must be 1-D."
    )


if labels.ndim != 1:

    raise RuntimeError(
        "labels must be 1-D."
    )


if packet_lengths.ndim != 2:

    raise RuntimeError(
        "packet_lengths must be 2-D."
    )


if labels.size != EXPECTED_FLOW_COUNT:

    raise RuntimeError(
        "Unexpected Monday corpus flow count."
    )


if flow_offsets.size != (
    EXPECTED_FLOW_COUNT
    +
    1
):

    raise RuntimeError(
        "flow_offsets must contain N_flows + 1 boundaries."
    )


if packet_lengths.shape != (
    EXPECTED_FLOW_COUNT,
    64,
):

    raise RuntimeError(
        "Unexpected packet_lengths geometry."
    )


print(
    "\n[PASS] Structural geometry:"
)

print(
    f"  flows             : {EXPECTED_FLOW_COUNT:,}"
)

print(
    f"  byte boundaries   : {flow_offsets.size:,}"
)

print(
    f"  packet slots/flow : {packet_lengths.shape[1]}"
)


# =============================================================================
# 7. BYTE-OFFSET SEMANTICS
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY RECOVERY :: FLOW BYTE OFFSETS"
)


offsets_monotonic = bool(
    np.all(
        flow_offsets[
            1:
        ]
        >=
        flow_offsets[
            :-1
        ]
    )
)


offsets_strict = bool(
    np.all(
        flow_offsets[
            1:
        ]
        >
        flow_offsets[
            :-1
        ]
    )
)


first_offset = int(
    flow_offsets[
        0
    ]
)

last_offset = int(
    flow_offsets[
        -1
    ]
)


flow_byte_lengths = (
    np.diff(
        flow_offsets.astype(
            np.int64,
            copy=False,
        )
    )
)


print(
    "First offset:",
    first_offset
)

print(
    "Last offset :",
    last_offset
)

print(
    "Expected encoded_bytes.bin bytes:",
    EXPECTED_ENCODED_BYTES
)

print(
    "Offsets nondecreasing:",
    offsets_monotonic
)

print(
    "Offsets strictly increasing:",
    offsets_strict
)


if first_offset != 0:

    raise RuntimeError(
        "First flow byte offset must be zero."
    )


if last_offset != EXPECTED_ENCODED_BYTES:

    raise RuntimeError(
        "Final flow offset does not equal encoded_bytes.bin length."
    )


if not offsets_strict:

    raise RuntimeError(
        "Flow byte boundaries are not strictly increasing."
    )


if np.any(
    flow_byte_lengths <= 0
):

    raise RuntimeError(
        "Zero/negative encoded flow byte length found."
    )


print(
    "\nEncoded flow byte lengths:"
)

print(
    "  min   :",
    int(
        flow_byte_lengths.min()
    )
)

print(
    "  max   :",
    int(
        flow_byte_lengths.max()
    )
)

print(
    "  median:",
    float(
        np.median(
            flow_byte_lengths
        )
    )
)

print(
    "  mean  :",
    float(
        flow_byte_lengths.mean()
    )
)


# =============================================================================
# 8. PACKET-LENGTH MATRIX SEMANTICS
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY RECOVERY :: PACKET LENGTH MATRIX"
)


packet_lengths_nonnegative = bool(
    np.all(
        packet_lengths
        >=
        0
    )
)


packet_lengths_max = int(
    packet_lengths.max()
)


print(
    "All packet lengths >= 0:",
    packet_lengths_nonnegative
)

print(
    "Maximum packet length    :",
    packet_lengths_max
)


if not packet_lengths_nonnegative:

    raise RuntimeError(
        "Negative packet length found."
    )


if packet_lengths_max > 256:

    raise RuntimeError(
        "Packet length exceeds frozen 256-byte representation cap."
    )


# Zero is candidate padding.
nonzero_mask = (
    packet_lengths
    >
    0
)


packet_counts = np.count_nonzero(
    nonzero_mask,
    axis=1,
)


row_packet_byte_sums = packet_lengths.astype(
    np.uint64,
    copy=False,
).sum(
    axis=1,
    dtype=np.uint64,
)


print(
    "\nDerived packet counts:"
)

print(
    "  min packets/flow:",
    int(
        packet_counts.min()
    )
)

print(
    "  max packets/flow:",
    int(
        packet_counts.max()
    )
)

print(
    "  median          :",
    float(
        np.median(
            packet_counts
        )
    )
)

print(
    "  mean            :",
    float(
        packet_counts.mean()
    )
)

print(
    "  flows with 0    :",
    int(
        np.count_nonzero(
            packet_counts == 0
        )
    )
)

print(
    "  flows with 1    :",
    int(
        np.count_nonzero(
            packet_counts == 1
        )
    )
)

print(
    "  flows with 64   :",
    int(
        np.count_nonzero(
            packet_counts == 64
        )
    )
)


# =============================================================================
# 9. ZERO-PADDING CONTIGUITY TEST
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY RECOVERY :: ZERO-PADDING SEMANTICS"
)


# For each row, once zero-padding begins, no later positive packet length
# should appear.
padding_violation_rows = 0


for col in range(
    1,
    packet_lengths.shape[
        1
    ],
):

    earlier_zero = (
        packet_lengths[
            :,
            col - 1
        ]
        ==
        0
    )

    later_positive = (
        packet_lengths[
            :,
            col
        ]
        >
        0
    )


    padding_violation_rows += int(
        np.count_nonzero(
            earlier_zero
            &
            later_positive
        )
    )


print(
    "Zero -> later-positive violations:",
    padding_violation_rows
)


if padding_violation_rows != 0:

    raise RuntimeError(
        "packet_lengths zero-padding is not contiguous."
    )


print(
    "[PASS] Zeros form trailing padding only."
)


# =============================================================================
# 10. CRITICAL CROSS-ARRAY CONSISTENCY TEST
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY RECOVERY :: BYTE CONSISTENCY"
)


# Hypothesis:
#
#   diff(flow_offsets)[i]
#       ==
#   sum(packet_lengths[i, :])
#
# If this holds for every flow, then:
#
#   flow_offsets = boundaries into encoded_bytes.bin
#   packet_lengths = 64-slot zero-padded packet lengths for each flow
#   encoded bytes for flow i occupy:
#
#       encoded_bytes[
#           flow_offsets[i] : flow_offsets[i+1]
#       ]
#

byte_match = (
    row_packet_byte_sums
    ==
    flow_byte_lengths.astype(
        np.uint64,
        copy=False,
    )
)


matching_flows = int(
    np.count_nonzero(
        byte_match
    )
)


mismatching_flows = int(
    byte_match.size
    -
    matching_flows
)


print(
    "Flows tested     :",
    f"{byte_match.size:,}"
)

print(
    "Exact byte match :",
    f"{matching_flows:,}"
)

print(
    "Mismatch         :",
    f"{mismatching_flows:,}"
)


if mismatching_flows:

    bad = np.flatnonzero(
        ~byte_match
    )


    print(
        "\nFirst mismatching indices:"
    )


    for idx in bad[
        :20
    ]:

        print(
            f"  flow {int(idx):,}: "
            f"offset_delta={int(flow_byte_lengths[idx])} "
            f"packet_sum={int(row_packet_byte_sums[idx])}"
        )


    raise RuntimeError(
        "packet_lengths row sums do not exactly reproduce flow byte boundaries."
    )


print(
    "\n[PASS] ALL flows satisfy:"
)

print(
    "  diff(flow_offsets) == sum(packet_lengths, axis=1)"
)


# =============================================================================
# 11. GLOBAL BYTE CONSISTENCY
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY RECOVERY :: GLOBAL BYTE ACCOUNTING"
)


total_packet_bytes = int(
    row_packet_byte_sums.sum(
        dtype=np.uint64
    )
)


print(
    "Sum of all packet lengths :",
    f"{total_packet_bytes:,}"
)

print(
    "encoded_bytes.bin bytes   :",
    f"{EXPECTED_ENCODED_BYTES:,}"
)

print(
    "final flow offset          :",
    f"{last_offset:,}"
)


if not (
    total_packet_bytes
    ==
    EXPECTED_ENCODED_BYTES
    ==
    last_offset
):

    raise RuntimeError(
        "Global encoded byte accounting mismatch."
    )


print(
    "\n[PASS] Global byte accounting is exact."
)


# =============================================================================
# 12. LABEL GEOMETRY
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY RECOVERY :: LABELS"
)


label_values, label_counts = np.unique(
    labels,
    return_counts=True,
)


for value, count in zip(
    label_values.tolist(),
    label_counts.tolist(),
):

    print(
        f"label {value}: {count:,}"
    )


labels_map = {
    str(
        int(
            value
        )
    ):
        int(
            count
        )
    for value, count in zip(
        label_values,
        label_counts,
    )
}


# =============================================================================
# 13. DISTRIBUTION COUNTS
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY RECOVERY :: REPRESENTATION DISTRIBUTION"
)


packet_count_values, packet_count_freq = np.unique(
    packet_counts,
    return_counts=True,
)


print(
    "Packet-count distribution:"
)


for count, freq in zip(
    packet_count_values.tolist(),
    packet_count_freq.tolist(),
):

    print(
        f"  {int(count):2d} packets : {int(freq):,} flows"
    )


# =============================================================================
# 14. FINAL RESOLVED SEMANTICS
# =============================================================================

resolved_semantics = {
    "flow_count":
        EXPECTED_FLOW_COUNT,

    "encoded_bytes_size":
        EXPECTED_ENCODED_BYTES,

    "flow_offsets_semantics":
        (
            "N+1 uint64 byte boundaries into encoded_bytes.bin; "
            "flow i occupies bytes [flow_offsets[i], flow_offsets[i+1])."
        ),

    "packet_lengths_semantics":
        (
            "N x 64 uint16 per-flow packet encoded lengths; "
            "positive lengths are contiguous from column 0; "
            "remaining columns are zero padding."
        ),

    "flow_packet_count_semantics":
        (
            "count_nonzero(packet_lengths[i]) gives retained encoded "
            "packet count for flow i, capped at 64."
        ),

    "cross_array_identity":
        (
            "For every flow i: "
            "flow_offsets[i+1] - flow_offsets[i] == "
            "sum(packet_lengths[i,:])."
        ),

    "labels_semantics":
        (
            "One uint8 label per corpus flow."
        ),

    "corpus_boundary":
        (
            "This Release corpus is already post raw-PCAP parsing, "
            "flow reconstruction, supervised flow matching/filtering, "
            "packet selection/truncation, and packet-byte masking/encoding."
        ),
}


banner(
    "MONDAY CORPUS GEOMETRY RECOVERY :: RESOLVED SEMANTICS"
)


for key, value in resolved_semantics.items():

    print(
        f"{key}:"
    )

    print(
        " ",
        value
    )


# =============================================================================
# 15. WRITE CORRECTED GEOMETRY RECEIPT
# =============================================================================

receipt = {
    "schema":
        "stage26_monday_release_corpus_geometry_audit_v2",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "scientific_parent":
        EXPECTED_HEAD,

    "status":
        "PASS_EXISTING_RELEASE_CORPUS_GEOMETRY_RESOLVED",

    "release": {
        "tag":
            "stage20-compact-corpora-v1",

        "asset":
            "stage20-Monday-compact-corpus-v1.tar",

        "tar_size_bytes":
            EXPECTED_TAR_SIZE,

        "tar_sha256":
            EXPECTED_TAR_SHA256,
    },

    "member_sha256":
        EXPECTED_MEMBER_SHA256,

    "geometry": {
        "flow_count":
            EXPECTED_FLOW_COUNT,

        "encoded_bytes_size":
            EXPECTED_ENCODED_BYTES,

        "flow_offsets": {
            "dtype":
                str(
                    flow_offsets.dtype
                ),

            "shape":
                list(
                    flow_offsets.shape
                ),

            "first":
                first_offset,

            "last":
                last_offset,

            "strictly_increasing":
                offsets_strict,
        },

        "labels": {
            "dtype":
                str(
                    labels.dtype
                ),

            "shape":
                list(
                    labels.shape
                ),

            "distribution":
                labels_map,
        },

        "packet_lengths": {
            "dtype":
                str(
                    packet_lengths.dtype
                ),

            "shape":
                list(
                    packet_lengths.shape
                ),

            "max_length":
                packet_lengths_max,

            "trailing_zero_padding_only":
                padding_violation_rows
                ==
                0,

            "min_packets_per_flow":
                int(
                    packet_counts.min()
                ),

            "max_packets_per_flow":
                int(
                    packet_counts.max()
                ),

            "mean_packets_per_flow":
                float(
                    packet_counts.mean()
                ),

            "median_packets_per_flow":
                float(
                    np.median(
                        packet_counts
                    )
                ),

            "flows_with_64_packets":
                int(
                    np.count_nonzero(
                        packet_counts
                        ==
                        64
                    )
                ),
        },

        "cross_array_validation": {
            "flows_checked":
                int(
                    byte_match.size
                ),

            "exact_matches":
                matching_flows,

            "mismatches":
                mismatching_flows,

            "total_packet_length_bytes":
                total_packet_bytes,

            "encoded_bytes_size":
                EXPECTED_ENCODED_BYTES,

            "final_flow_offset":
                last_offset,
        },
    },

    "resolved_semantics":
        resolved_semantics,

    "scientific_boundary": {
        "existing_release_corpus_authoritative":
            True,

        "corpus_recreated_from_pcap":
            False,

        "pcap_opened":
            False,

        "pcap_packets_iterated":
            False,

        "raw_extraction_performed":
            False,

        "timing_performed":
            False,

        "model_loaded":
            False,

        "inference":
            False,

        "Thursday_accessed":
            False,

        "Friday_accessed":
            False,

        "gpu_used":
            False,

        "git_modified":
            False,
    },

    "stage26_roles": {
        "raw_pcap":
            "RAW_CAPTURE_EXTRACTION_BENCHMARK_ONLY",

        "release_corpus":
            (
                "AUTHORITATIVE_EXISTING_DOWNSTREAM_CORPUS_FOR_"
                "REPRESENTATION_AND_PREPROCESSING_PROFILING"
            ),

        "corpus_regeneration_from_pcap":
            "FORBIDDEN",
    },
}


atomic_json(
    OUTPUT,
    receipt,
)


receipt_sha = sha256_file(
    OUTPUT
)


# =============================================================================
# 16. FINAL GIT AUDIT
# =============================================================================

banner(
    "MONDAY CORPUS GEOMETRY RECOVERY :: FINAL AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "Local HEAD :",
    final_head
)

print(
    "Remote HEAD:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_HEAD:

    raise RuntimeError(
        "HEAD changed."
    )


if final_remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed."
    )


if final_status:

    raise RuntimeError(
        "Repository changed."
    )


banner(
    "MONDAY RELEASE CORPUS GEOMETRY RESOLVED"
)


print(
    "FLOW COUNT:"
)

print(
    " ",
    f"{EXPECTED_FLOW_COUNT:,}"
)


print(
    "\nFLOW OFFSETS:"
)

print(
    "  boundaries:",
    f"{flow_offsets.size:,}"
)

print(
    "  first     :",
    first_offset
)

print(
    "  last      :",
    last_offset
)


print(
    "\nPACKET LENGTH MATRIX:"
)

print(
    "  shape:",
    packet_lengths.shape
)

print(
    "  dtype:",
    packet_lengths.dtype
)


print(
    "\nBYTE IDENTITY:"
)

print(
    "  diff(flow_offsets) == row packet-length sums:"
)

print(
    " ",
    f"{matching_flows:,}/{EXPECTED_FLOW_COUNT:,} exact"
)


print(
    "\nGLOBAL ENCODED BYTES:"
)

print(
    " ",
    f"{total_packet_bytes:,}"
)


print(
    "\nCORRECTED GEOMETRY RECEIPT:"
)

print(
    " ",
    OUTPUT
)

print(
    " SHA256:",
    receipt_sha
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  RELEASE CORPUS AUTHORITATIVE       : YES"
)

print(
    "  CORPUS RECREATED FROM PCAP         : NO"
)

print(
    "  PCAP ACCESSED                      : NO"
)

print(
    "  EXTRACTION                         : NO"
)

print(
    "  TIMING                             : NO"
)

print(
    "  GPU                                : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Freeze Stage26-4 with the corrected decomposition:"
)

print(
    "    A. raw PCAP -> flow reconstruction benchmark only"
)

print(
    "    B. frozen Release corpus -> representation benchmark"
)

print(
    "    C. representation -> inference already measured"
)


MONDAY CORPUS GEOMETRY RECOVERY :: SCIENTIFIC ANCHOR
Expected HEAD : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Local HEAD    : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
origin/main   : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Repo clean    : True

MONDAY CORPUS GEOMETRY RECOVERY :: UPSTREAM AUDIT
Expected structure audit SHA256:
  a0ed17fa8d26049f53766e32981dff72724d79fbd573e7aab79c926a36833c3b
Actual structure audit SHA256:
  a0ed17fa8d26049f53766e32981dff72724d79fbd573e7aab79c926a36833c3b

MONDAY CORPUS GEOMETRY RECOVERY :: TAR GATE
Expected TAR size: 595261440
Actual TAR size  : 595261440
Full TAR SHA256 previously verified by upstream audit:
  4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20

MONDAY CORPUS GEOMETRY RECOVERY :: LOAD ARRAYS
Monday/flow_offsets.npy
  SHA256: 3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e
  dtype : uint64
  shape : (528510,)
  size  : 528,510
Monday/labels.npy
  SHA256: 48792b8d6a127b35342cb0789baa6c54396f1100a60ce7225d

In [12]:
# =============================================================================
# STAGE26-4B1
# FREEZE RAW PCAP -> SOURCE-FAITHFUL FLOW RECONSTRUCTION PROTOCOL
#
# CURRENT DURABLE PARENT:
#   347d93f21d454cc5bda2c45889c890c67cdf0ecc
#
# ABSOLUTE RULES:
#   - DO NOT recreate any Stage20 compact corpus from the PCAP.
#   - DO NOT open/iterate the PCAP in this cell.
#   - DO NOT perform extraction in this cell.
#   - DO NOT perform timing in this cell.
#   - DO NOT access Thursday/Friday.
#   - DO NOT load models.
#   - GPU remains OFF.
#
# COMPONENT DECOMPOSITION:
#
#   A. RAW PCAP
#      -> IPv4/direct-transport parsing
#      -> source-faithful flow reconstruction
#      [THIS protocol]
#
#   B. EXISTING GitHub Release compact corpus
#      -> representation/materialization
#      [AUTHORITATIVE corpus; separate future protocol]
#
#   C. representation
#      -> inference
#      [already Stage26-2]
#
# IMPORTANT:
#   A and B are NOT assumed to be directly additive into a complete
#   end-to-end pipeline because the frozen release corpus already lies
#   downstream of matching/filtering/masking/truncation.
# =============================================================================

from __future__ import annotations

import os
import sys
import json
import hashlib
import shutil
import subprocess
import textwrap
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. FROZEN IDENTITIES / PATHS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

EXTRACTION_ROOT = (
    STAGE26_ROOT
    / "extraction"
)

WORKERS_ROOT = (
    STAGE26_ROOT
    / "workers"
)

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

WORKERS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


EXPECTED_PARENT = (
    "347d93f21d454cc5bda2c45889c890c67cdf0ecc"
)


# -----------------------------------------------------------------------------
# Original Stage26 protocol
# -----------------------------------------------------------------------------

MEASUREMENT_PROTOCOL = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXPECTED_MEASUREMENT_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)


# -----------------------------------------------------------------------------
# Historical Stage20 Monday raw geometry
# -----------------------------------------------------------------------------

GEOMETRY_PROFILE = (
    REPO
    / "results"
    / "stage20_1d_representation"
    / "stage20_1d2m_monday_exact_geometry_profile.json"
)

EXPECTED_GEOMETRY_SHA256 = (
    "3a26d6499334c12ea4e9272aef4250761c6cf4399e7fb4d33ef236f12d0b7272"
)


# -----------------------------------------------------------------------------
# Exact raw Monday source
# -----------------------------------------------------------------------------

MONDAY_PCAP = (
    STAGE26_ROOT
    / "sources"
    / "Monday-WorkingHours.pcap"
)

PCAP_FILENAME = (
    "Monday-WorkingHours.pcap"
)

PCAP_SIZE_BYTES = (
    10_822_507_416
)

PCAP_SHA256 = (
    "f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972"
)

PCAP_CONTAINER = (
    "PCAPNG"
)


# -----------------------------------------------------------------------------
# Latest source-restoration receipt.
#
# This SHA is from the user's most recent successful Stage26-4A rerun.
# -----------------------------------------------------------------------------

SOURCE_RECEIPT = (
    EXTRACTION_ROOT
    / "stage26_4a_monday_source_restoration_receipt.json"
)

EXPECTED_SOURCE_RECEIPT_SHA256 = (
    "a7d25f504c5a1d75f6cb46b94bd498ae6ca236a3872be5f868ce9160ec269bb3"
)


# -----------------------------------------------------------------------------
# Extractor/runtime inventory
# -----------------------------------------------------------------------------

EXTRACTOR_INVENTORY = (
    EXTRACTION_ROOT
    / "stage26_4b0_extractor_runtime_inventory.json"
)

EXPECTED_EXTRACTOR_INVENTORY_SHA256 = (
    "8e8188f2606dfbfdc7d662a378df29b17c0540416d061417d288b02ff7a0f686"
)


# -----------------------------------------------------------------------------
# Existing Release corpus audits
# -----------------------------------------------------------------------------

CORPUS_STRUCTURE_AUDIT = (
    EXTRACTION_ROOT
    / "stage26_monday_release_corpus_structure_audit.json"
)

EXPECTED_CORPUS_STRUCTURE_SHA256 = (
    "a0ed17fa8d26049f53766e32981dff72724d79fbd573e7aab79c926a36833c3b"
)

CORPUS_GEOMETRY_AUDIT = (
    EXTRACTION_ROOT
    / "stage26_monday_release_corpus_geometry_audit.json"
)

EXPECTED_CORPUS_GEOMETRY_SHA256 = (
    "47ad682502cfe3ca98d794e9a800263fed1b1c8b5331f5400160d27a518d129a"
)


# -----------------------------------------------------------------------------
# Frozen Release corpus identity
# -----------------------------------------------------------------------------

RELEASE_TAG = (
    "stage20-compact-corpora-v1"
)

MONDAY_CORPUS_ASSET = (
    "stage20-Monday-compact-corpus-v1.tar"
)

MONDAY_CORPUS_SIZE = (
    595_261_440
)

MONDAY_CORPUS_SHA256 = (
    "4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20"
)

MONDAY_CORPUS_FLOW_COUNT = (
    528_509
)

MONDAY_CORPUS_ENCODED_BYTES = (
    522_845_159
)

MONDAY_CORPUS_MEMBER_SHA256 = {
    "encoded_bytes.bin":
        "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",

    "flow_offsets.npy":
        "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",

    "labels.npy":
        "48792b8d6a127b35342cb0789baa6c54396f1100a60ce7225daf08d1c3530424",

    "packet_lengths.npy":
        "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
}


# =============================================================================
# 1. HISTORICAL FULL-MONDAY RAW CORRECTNESS ANCHORS
# =============================================================================

RAW_PACKET_COUNT = (
    11_709_971
)

VALID_IPV4_PACKET_COUNT = (
    11_626_492
)

NON_IPV4_PACKET_COUNT = (
    83_479
)

PARSER_TCP_COUNT = (
    10_718_469
)

PARSER_UDP_COUNT = (
    907_039
)

PARSER_OTHER0_COUNT = (
    984
)

RAW_EXPORTABLE_FLOW_COUNT = (
    529_601
)

RETAINED_PACKET_COUNT = (
    11_573_331
)

MAX_ACTIVE_FLOWS = (
    246_086
)

FIN_FLOW_COUNT = (
    216_388
)

TIMEOUT_FLOW_COUNT = (
    120_017
)

EOF_CURRENT_FLOW_COUNT = (
    193_196
)

TIMEOUT_SINGLETON_DISCARDED = (
    271
)

EOF_SINGLETON_DISCARDED = (
    52_890
)


# -----------------------------------------------------------------------------
# Frozen parser / lifecycle identities
# -----------------------------------------------------------------------------

PARSER_SEMANTICS_ID = (
    "STAGE20_FROZEN_IPV4_DIRECT_TRANSPORT_TCP6_UDP17_OTHER0"
)

FLOW_ID_SEMANTICS_ID = (
    "BASICPACKETINFO_JAVA_SIGNED_IPV4_FIRST_DIFFERING_BYTE_SWAP_IPS_AND_PORTS_TOGETHER"
)

FLOW_LIFECYCLE_ID = (
    "CICFLOWMETER_FLOWGENERATOR_EAA853DD82F08BA5288BB7F295B471DE7313F883"
)

CICFLOWMETER_COMMIT = (
    "eaa853dd82f08ba5288bb7f295b471de7313f883"
)

CICFLOWMETER_TREE = (
    "56a11f8fecab86228739fbe3259ff34f3bc841f4"
)

FLOW_TIMEOUT_US = (
    120_000_000
)


# =============================================================================
# 2. FROZEN TIMED EXTRACTION SAMPLE
# =============================================================================

# Pre-result deterministic bounded sequential sample.
#
# This value is frozen BEFORE any extraction performance measurement.
# It is not selected from Stage26 model latency or memory results.

TIMED_RAW_PACKET_START = (
    1
)

TIMED_RAW_PACKET_END = (
    1_000_000
)

TIMED_RAW_PACKET_COUNT = (
    1_000_000
)

TIMED_SAMPLE_FRACTION = (
    TIMED_RAW_PACKET_COUNT
    /
    RAW_PACKET_COUNT
)


# Primary CPU execution condition.
CPU_MODE = (
    "CPU_1_PHYSICAL_CORE"
)

CPU_AFFINITY = [
    0
]

CPU_THREAD_COUNT = (
    1
)

TIMED_REPETITIONS = (
    5
)

RESOURCE_TIMEOUT_SECONDS = (
    600
)


# Stage26 environment gate reused unchanged.
CPU_UTILIZATION_MAX_PERCENT = (
    20.0
)

AVAILABLE_RAM_MIN_GIB = (
    8.0
)

ENV_GATE_RETRIES = (
    3
)

ENV_GATE_RETRY_COOLDOWN_SECONDS = (
    5
)

CPU_UTILIZATION_SAMPLE_SECONDS = (
    1.0
)


# =============================================================================
# 3. OUTPUT PATHS
# =============================================================================

WORKER_PATH = (
    WORKERS_ROOT
    / "stage26_raw_flow_extractor_v1.py"
)

PROTOCOL_PATH = (
    EXTRACTION_ROOT
    / "stage26_extraction_protocol.json"
)

BOUNDARY_PATH = (
    EXTRACTION_ROOT
    / "stage26_component_boundary_map.json"
)

FREEZE_RECEIPT = (
    EXTRACTION_ROOT
    / "stage26_4b1_extraction_protocol_freeze_receipt.json"
)


REPO_LOCK_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4b_extraction_protocol_lock"
)


# =============================================================================
# 4. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 120
    )

    print(text)

    print(
        "=" * 120
    )


def run(
    cmd,
    *,
    cwd=None,
    check=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )


    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            + " ".join(
                map(
                    str,
                    cmd,
                )
            )
            + "\n\n"
            + p.stdout
        )


    return p


def git(*args):

    return run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        check=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()


    with Path(path).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )


            if not chunk:

                break


            h.update(
                chunk
            )


    return h.hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )


    os.replace(
        tmp,
        path,
    )


# =============================================================================
# 5. SCIENTIFIC ANCHOR
# =============================================================================

banner(
    "STAGE26-4B1 :: SCIENTIFIC ANCHOR"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status_before = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status_before == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected local Stage26 parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected origin/main."
    )


if status_before:

    raise RuntimeError(
        "Repository must be clean before freezing Stage26-4B1."
    )


# =============================================================================
# 6. UPSTREAM PROVENANCE HASH GATE
# =============================================================================

banner(
    "STAGE26-4B1 :: UPSTREAM PROVENANCE GATE"
)


hash_gates = [
    (
        "Stage26 measurement protocol",
        MEASUREMENT_PROTOCOL,
        EXPECTED_MEASUREMENT_PROTOCOL_SHA256,
    ),
    (
        "Stage20 Monday geometry",
        GEOMETRY_PROFILE,
        EXPECTED_GEOMETRY_SHA256,
    ),
    (
        "Stage26-4A source receipt",
        SOURCE_RECEIPT,
        EXPECTED_SOURCE_RECEIPT_SHA256,
    ),
    (
        "Stage26-4B0 extractor inventory",
        EXTRACTOR_INVENTORY,
        EXPECTED_EXTRACTOR_INVENTORY_SHA256,
    ),
    (
        "Monday corpus structure audit",
        CORPUS_STRUCTURE_AUDIT,
        EXPECTED_CORPUS_STRUCTURE_SHA256,
    ),
    (
        "Monday corpus geometry audit",
        CORPUS_GEOMETRY_AUDIT,
        EXPECTED_CORPUS_GEOMETRY_SHA256,
    ),
]


for name, path, expected in hash_gates:

    if not path.exists():

        raise FileNotFoundError(
            path
        )


    actual = sha256_file(
        path
    )


    passed = (
        actual
        ==
        expected
    )


    print(
        f"{name:37s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Frozen provenance mismatch: {name}"
        )


# =============================================================================
# 7. ORIGINAL STAGE26 EXTRACTION GATE
# =============================================================================

banner(
    "STAGE26-4B1 :: ORIGINAL EXTRACTION GATE"
)


measurement_protocol = json.loads(
    MEASUREMENT_PROTOCOL.read_text(
        encoding="utf-8"
    )
)


original_gate = measurement_protocol[
    "extraction_protocol_gate"
]


print(
    json.dumps(
        original_gate,
        indent=2,
        sort_keys=True,
    )
)


if (
    original_gate[
        "pcap_required_before_extraction_timing"
    ]
    is not True
):

    raise RuntimeError(
        "Original PCAP requirement changed."
    )


if (
    original_gate[
        "inference_results_may_select_pcap"
    ]
    is not False
):

    raise RuntimeError(
        "Original source-selection rule changed."
    )


# =============================================================================
# 8. SOURCE GATE — NO PCAP OPEN / HASH
# =============================================================================

banner(
    "STAGE26-4B1 :: RAW SOURCE PRESENCE GATE"
)


if not MONDAY_PCAP.exists():

    raise FileNotFoundError(
        MONDAY_PCAP
    )


resolved_pcap = MONDAY_PCAP.resolve(
    strict=True
)


pcap_size_actual = int(
    resolved_pcap.stat().st_size
)


source_receipt = json.loads(
    SOURCE_RECEIPT.read_text(
        encoding="utf-8"
    )
)


receipt_source_sha = (
    source_receipt[
        "recovery"
    ][
        "verified_sha256"
    ]
)


print(
    "Canonical source:"
)

print(
    " ",
    MONDAY_PCAP
)

print(
    "Resolved source:"
)

print(
    " ",
    resolved_pcap
)

print(
    "Size:",
    pcap_size_actual
)

print(
    "Previously byte-verified SHA256:"
)

print(
    " ",
    receipt_source_sha
)


if pcap_size_actual != PCAP_SIZE_BYTES:

    raise RuntimeError(
        "Monday source byte size changed."
    )


if receipt_source_sha != PCAP_SHA256:

    raise RuntimeError(
        "Monday source identity mismatch."
    )


print(
    "\nPCAP opened by this cell: NO"
)


# =============================================================================
# 9. VERIFY RAW HISTORICAL GEOMETRY / PROVENANCE
# =============================================================================

banner(
    "STAGE26-4B1 :: HISTORICAL RAW GEOMETRY GATE"
)


historical = json.loads(
    GEOMETRY_PROFILE.read_text(
        encoding="utf-8"
    )
)


reconstruction = historical[
    "reconstruction_provenance"
]


required_semantics = {
    "parser_semantics_identifier":
        PARSER_SEMANTICS_ID,

    "flow_id_semantics_identifier":
        FLOW_ID_SEMANTICS_ID,

    "flow_lifecycle_semantics_identifier":
        FLOW_LIFECYCLE_ID,

    "flow_timeout_us":
        FLOW_TIMEOUT_US,
}


for key, expected in required_semantics.items():

    actual = reconstruction[
        key
    ]


    print(
        f"{key:40s}:",
        actual
    )


    if actual != expected:

        raise RuntimeError(
            f"Historical semantics mismatch: {key}"
        )


# =============================================================================
# 10. VERIFY EXISTING RELEASE CORPUS BOUNDARY
# =============================================================================

banner(
    "STAGE26-4B1 :: AUTHORITATIVE RELEASE CORPUS BOUNDARY"
)


corpus_geometry = json.loads(
    CORPUS_GEOMETRY_AUDIT.read_text(
        encoding="utf-8"
    )
)


if (
    corpus_geometry[
        "status"
    ]
    !=
    "PASS_EXISTING_RELEASE_CORPUS_GEOMETRY_RESOLVED"
):

    raise RuntimeError(
        "Existing Release corpus geometry audit is not PASS."
    )


if (
    corpus_geometry[
        "geometry"
    ][
        "flow_count"
    ]
    !=
    MONDAY_CORPUS_FLOW_COUNT
):

    raise RuntimeError(
        "Release corpus flow count changed."
    )


print(
    "Release tag:"
)

print(
    " ",
    RELEASE_TAG
)

print(
    "Monday asset:"
)

print(
    " ",
    MONDAY_CORPUS_ASSET
)

print(
    "Release flow count:"
)

print(
    " ",
    f"{MONDAY_CORPUS_FLOW_COUNT:,}"
)

print(
    "Raw exportable flow count:"
)

print(
    " ",
    f"{RAW_EXPORTABLE_FLOW_COUNT:,}"
)

print(
    "\nThese counts are intentionally NOT required to match."
)

print(
    "The Release corpus is downstream of supervised matching/filtering."
)

print(
    "\nCORPUS REGENERATION FROM PCAP: FORBIDDEN"
)


# =============================================================================
# 11. FREEZE SOURCE-FAITHFUL RAW EXTRACTION WORKER
# =============================================================================

banner(
    "STAGE26-4B1 :: WRITE RAW EXTRACTION WORKER"
)


worker_source = r'''
from __future__ import annotations

import os
import sys
import json
import struct
import hashlib
import time
from pathlib import Path


# =============================================================================
# FROZEN SEMANTICS
# =============================================================================

FLOW_TIMEOUT_US = 120_000_000

PCAPNG_SHB = 0x0A0D0D0A
PCAPNG_IDB = 0x00000001
PCAPNG_PB  = 0x00000002
PCAPNG_SPB = 0x00000003
PCAPNG_EPB = 0x00000006

LINKTYPE_ETHERNET = 1

VLAN_ETHERTYPES = {
    0x8100,
    0x88A8,
    0x9100,
}


# =============================================================================
# OUTPUT
# =============================================================================

def atomic_json(path, obj):

    path = Path(path)

    tmp = Path(
        str(path)
        + ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write("\n")

        f.flush()

        os.fsync(
            f.fileno()
        )


    os.replace(
        tmp,
        path,
    )


# =============================================================================
# EXACT BASICPACKETINFO FLOW-ID SEMANTICS
# =============================================================================

def java_signed_byte(value):

    return (
        value
        if value < 128
        else value - 256
    )


def canonical_flow_key(
    src,
    dst,
    src_port,
    dst_port,
    protocol,
):

    # BasicPacketInfo.generateFlowId():
    #
    # Compare source/destination address bytes using Java signed Byte
    # comparison. First differing byte decides direction. Ports swap
    # together with IP addresses.
    forward = True


    for s, d in zip(
        src,
        dst,
    ):

        ss = java_signed_byte(
            s
        )

        dd = java_signed_byte(
            d
        )


        if ss != dd:

            if ss > dd:

                forward = False

            break


    if forward:

        return (
            src,
            dst,
            int(src_port),
            int(dst_port),
            int(protocol),
        )


    return (
        dst,
        src,
        int(dst_port),
        int(src_port),
        int(protocol),
    )


# =============================================================================
# PCAPNG OPTIONS / TIMESTAMPS
# =============================================================================

def parse_options(
    data,
    endian,
):

    offset = 0

    result = []


    while (
        offset + 4
        <=
        len(data)
    ):

        code, length = struct.unpack_from(
            endian + "HH",
            data,
            offset,
        )

        offset += 4


        if code == 0:

            break


        if (
            offset + length
            >
            len(data)
        ):

            raise RuntimeError(
                "Malformed PCAPNG option."
            )


        value = data[
            offset:
            offset + length
        ]


        result.append(
            (
                int(code),
                value,
            )
        )


        offset += (
            length + 3
        ) & ~3


    return result


def interface_timestamp_config(
    options,
    endian,
):

    # PCAPNG default:
    # timestamp unit = 10^-6 second.
    resolution_kind = (
        "DECIMAL"
    )

    resolution_power = (
        6
    )

    offset_seconds = (
        0
    )


    for code, value in options:

        if (
            code == 9
            and
            len(value) >= 1
        ):

            raw = int(
                value[0]
            )


            if raw & 0x80:

                resolution_kind = (
                    "BINARY"
                )

                resolution_power = (
                    raw
                    &
                    0x7F
                )


            else:

                resolution_kind = (
                    "DECIMAL"
                )

                resolution_power = (
                    raw
                )


        elif (
            code == 14
            and
            len(value) >= 8
        ):

            offset_seconds = int(
                struct.unpack_from(
                    endian + "q",
                    value,
                    0,
                )[0]
            )


    return {
        "kind":
            resolution_kind,

        "power":
            int(
                resolution_power
            ),

        "offset_seconds":
            int(
                offset_seconds
            ),
    }


def timestamp_to_us(
    raw_timestamp,
    config,
):

    raw_timestamp = int(
        raw_timestamp
    )

    power = int(
        config[
            "power"
        ]
    )


    if (
        config[
            "kind"
        ]
        ==
        "DECIMAL"
    ):

        if power <= 6:

            us = (
                raw_timestamp
                *
                10 ** (
                    6 - power
                )
            )


        else:

            us = (
                raw_timestamp
                //
                10 ** (
                    power - 6
                )
            )


    else:

        us = (
            raw_timestamp
            *
            1_000_000
        ) // (
            1 << power
        )


    return (
        int(
            config[
                "offset_seconds"
            ]
        )
        *
        1_000_000
        +
        int(us)
    )


# =============================================================================
# ETHERNET -> FROZEN IPV4/DIRECT-TRANSPORT PARSER
# =============================================================================

def parse_ipv4_packet(
    frame,
):

    frame_len = len(
        frame
    )


    if frame_len < 14:

        return None


    ethertype = struct.unpack_from(
        ">H",
        frame,
        12,
    )[0]


    ipv4_offset = (
        14
    )


    while ethertype in VLAN_ETHERTYPES:

        if (
            ipv4_offset + 4
            >
            frame_len
        ):

            return None


        ethertype = struct.unpack_from(
            ">H",
            frame,
            ipv4_offset + 2,
        )[0]


        ipv4_offset += 4


    if ethertype != 0x0800:

        return None


    if (
        ipv4_offset + 20
        >
        frame_len
    ):

        return None


    version_ihl = int(
        frame[
            ipv4_offset
        ]
    )


    version = (
        version_ihl >> 4
    )

    ihl_bytes = (
        version_ihl
        &
        0x0F
    ) * 4


    if (
        version != 4
        or
        ihl_bytes < 20
        or
        ipv4_offset + ihl_bytes > frame_len
    ):

        return None


    src = bytes(
        frame[
            ipv4_offset + 12:
            ipv4_offset + 16
        ]
    )

    dst = bytes(
        frame[
            ipv4_offset + 16:
            ipv4_offset + 20
        ]
    )


    ip_protocol = int(
        frame[
            ipv4_offset + 9
        ]
    )


    fragment_field = struct.unpack_from(
        ">H",
        frame,
        ipv4_offset + 6,
    )[0]


    fragment_offset = (
        fragment_field
        &
        0x1FFF
    )


    transport_offset = (
        ipv4_offset
        +
        ihl_bytes
    )


    src_port = 0
    dst_port = 0

    normalized_protocol = (
        0
    )

    fin = (
        False
    )


    if (
        fragment_offset == 0
        and
        ip_protocol == 6
        and
        transport_offset + 20 <= frame_len
    ):

        src_port, dst_port = struct.unpack_from(
            ">HH",
            frame,
            transport_offset,
        )


        flags = int(
            frame[
                transport_offset + 13
            ]
        )


        fin = bool(
            flags
            &
            0x01
        )


        normalized_protocol = (
            6
        )


    elif (
        fragment_offset == 0
        and
        ip_protocol == 17
        and
        transport_offset + 8 <= frame_len
    ):

        src_port, dst_port = struct.unpack_from(
            ">HH",
            frame,
            transport_offset,
        )


        normalized_protocol = (
            17
        )


    # Stage20 geometry:
    # captured bytes begin at IPv4 header byte 0 and continue through
    # the end of the captured frame; Ethernet/VLAN bytes excluded.
    captured_ipv4_bytes = (
        frame_len
        -
        ipv4_offset
    )


    return {
        "key":
            canonical_flow_key(
                src,
                dst,
                src_port,
                dst_port,
                normalized_protocol,
            ),

        "protocol":
            normalized_protocol,

        "fin":
            fin,

        "captured_ipv4_bytes":
            int(
                captured_ipv4_bytes
            ),
    }


# =============================================================================
# EXACT CICFLOWMETER FLOWGENERATOR LIFECYCLE
# =============================================================================

def add_packet_to_flows(
    active,
    packet,
    timestamp_us,
    counters,
):

    key = packet[
        "key"
    ]


    flow = active.get(
        key
    )


    if flow is not None:

        # Exact FlowGenerator ordering:
        #
        #   IF timeout:
        #       export/discard OLD flow;
        #       current packet starts a NEW flow;
        #       FIN is NOT re-evaluated for replacement flow.
        #
        #   ELSE IF FIN:
        #       add current packet;
        #       export;
        #       remove.
        #
        #   ELSE:
        #       add current packet.

        if (
            int(timestamp_us)
            -
            flow[
                "start_us"
            ]
            >
            FLOW_TIMEOUT_US
        ):

            if flow[
                "packet_count"
            ] > 1:

                counters[
                    "timeout_exportable_flows"
                ] += 1

                counters[
                    "finished_exportable_flows"
                ] += 1

                counters[
                    "retained_packet_count_finished"
                ] += int(
                    flow[
                        "packet_count"
                    ]
                )


            else:

                counters[
                    "discarded_singleton_timeout_flows"
                ] += 1


            active[
                key
            ] = {
                "start_us":
                    int(
                        timestamp_us
                    ),

                "packet_count":
                    1,
            }


        elif packet[
            "fin"
        ]:

            flow[
                "packet_count"
            ] += 1


            counters[
                "fin_exportable_flows"
            ] += 1

            counters[
                "finished_exportable_flows"
            ] += 1

            counters[
                "retained_packet_count_finished"
            ] += int(
                flow[
                    "packet_count"
                ]
            )


            del active[
                key
            ]


        else:

            flow[
                "packet_count"
            ] += 1


    else:

        # Exact FlowGenerator behavior:
        # a first packet starts a flow even if it itself carries FIN.
        active[
            key
        ] = {
            "start_us":
                int(
                    timestamp_us
                ),

            "packet_count":
                1,
        }


    active_count = len(
        active
    )


    if (
        active_count
        >
        counters[
            "max_active_flows"
        ]
    ):

        counters[
            "max_active_flows"
        ] = int(
            active_count
        )


# =============================================================================
# SOURCE-FAITHFUL PCAPNG ITERATION
# =============================================================================

def extract(
    pcap_path,
    *,
    packet_limit=None,
):

    active = {}


    counters = {
        "raw_packet_count":
            0,

        "valid_ipv4_packet_count":
            0,

        "non_ipv4_packet_count":
            0,

        "parser_tcp_count":
            0,

        "parser_udp_count":
            0,

        "parser_other0_count":
            0,

        "captured_frame_bytes":
            0,

        "captured_ipv4_bytes":
            0,

        "fin_exportable_flows":
            0,

        "timeout_exportable_flows":
            0,

        "finished_exportable_flows":
            0,

        "discarded_singleton_timeout_flows":
            0,

        "retained_packet_count_finished":
            0,

        "max_active_flows":
            0,

        "eof_current_exportable_flows":
            0,

        "eof_singleton_discarded_flows":
            0,

        "retained_packet_count_eof":
            0,

        "unsupported_linktype_packet_count":
            0,

        "unsupported_simple_packet_blocks":
            0,

        "pcap_file_bytes_consumed":
            0,
    }


    endian = None

    interfaces = []


    with Path(
        pcap_path
    ).open(
        "rb"
    ) as f:

        while True:

            first8 = f.read(
                8
            )


            if not first8:

                break


            if len(
                first8
            ) != 8:

                raise RuntimeError(
                    "Truncated PCAPNG block header."
                )


            # -------------------------------------------------------------
            # SECTION HEADER BLOCK
            # -------------------------------------------------------------

            if (
                first8[
                    :4
                ]
                ==
                b"\x0a\x0d\x0d\x0a"
            ):

                bom = f.read(
                    4
                )


                if len(
                    bom
                ) != 4:

                    raise RuntimeError(
                        "Truncated PCAPNG SHB."
                    )


                if (
                    bom
                    ==
                    b"\x4d\x3c\x2b\x1a"
                ):

                    endian = "<"


                elif (
                    bom
                    ==
                    b"\x1a\x2b\x3c\x4d"
                ):

                    endian = ">"


                else:

                    raise RuntimeError(
                        "Unknown PCAPNG byte-order magic."
                    )


                total_length = struct.unpack(
                    endian + "I",
                    first8[
                        4:8
                    ],
                )[0]


                if total_length < 28:

                    raise RuntimeError(
                        "Invalid PCAPNG SHB length."
                    )


                remaining = (
                    total_length
                    -
                    12
                )


                rest = f.read(
                    remaining
                )


                if len(
                    rest
                ) != remaining:

                    raise RuntimeError(
                        "Truncated SHB."
                    )


                trailing = struct.unpack(
                    endian + "I",
                    rest[
                        -4:
                    ],
                )[0]


                if trailing != total_length:

                    raise RuntimeError(
                        "SHB length footer mismatch."
                    )


                interfaces = []

                continue


            if endian is None:

                raise RuntimeError(
                    "PCAPNG block before Section Header."
                )


            block_type = struct.unpack(
                endian + "I",
                first8[
                    :4
                ],
            )[0]


            total_length = struct.unpack(
                endian + "I",
                first8[
                    4:8
                ],
            )[0]


            if (
                total_length < 12
                or
                total_length % 4 != 0
            ):

                raise RuntimeError(
                    "Invalid PCAPNG block length."
                )


            remaining = (
                total_length
                -
                8
            )


            rest = f.read(
                remaining
            )


            if len(
                rest
            ) != remaining:

                raise RuntimeError(
                    "Truncated PCAPNG block."
                )


            trailing = struct.unpack(
                endian + "I",
                rest[
                    -4:
                ],
            )[0]


            if trailing != total_length:

                raise RuntimeError(
                    "PCAPNG block footer mismatch."
                )


            body = rest[
                :-4
            ]


            # -------------------------------------------------------------
            # INTERFACE DESCRIPTION BLOCK
            # -------------------------------------------------------------

            if block_type == PCAPNG_IDB:

                if len(
                    body
                ) < 8:

                    raise RuntimeError(
                        "Malformed IDB."
                    )


                linktype = struct.unpack_from(
                    endian + "H",
                    body,
                    0,
                )[0]


                snaplen = struct.unpack_from(
                    endian + "I",
                    body,
                    4,
                )[0]


                options = parse_options(
                    body[
                        8:
                    ],
                    endian,
                )


                ts_config = interface_timestamp_config(
                    options,
                    endian,
                )


                interfaces.append(
                    {
                        "linktype":
                            int(
                                linktype
                            ),

                        "snaplen":
                            int(
                                snaplen
                            ),

                        "timestamp":
                            ts_config,
                    }
                )


                continue


            # -------------------------------------------------------------
            # ENHANCED PACKET BLOCK
            # -------------------------------------------------------------

            if block_type == PCAPNG_EPB:

                if len(
                    body
                ) < 20:

                    raise RuntimeError(
                        "Malformed EPB."
                    )


                (
                    interface_id,
                    ts_high,
                    ts_low,
                    captured_length,
                    original_length,
                ) = struct.unpack_from(
                    endian + "IIIII",
                    body,
                    0,
                )


                if interface_id >= len(
                    interfaces
                ):

                    raise RuntimeError(
                        "EPB references unknown interface."
                    )


                if (
                    20 + captured_length
                    >
                    len(body)
                ):

                    raise RuntimeError(
                        "EPB captured length exceeds block."
                    )


                frame = body[
                    20:
                    20 + captured_length
                ]


                interface = interfaces[
                    interface_id
                ]


                raw_timestamp = (
                    (
                        int(ts_high)
                        <<
                        32
                    )
                    |
                    int(ts_low)
                )


                timestamp_us = timestamp_to_us(
                    raw_timestamp,
                    interface[
                        "timestamp"
                    ],
                )


            # -------------------------------------------------------------
            # OBSOLETE PACKET BLOCK
            # -------------------------------------------------------------

            elif block_type == PCAPNG_PB:

                if len(
                    body
                ) < 20:

                    raise RuntimeError(
                        "Malformed Packet Block."
                    )


                (
                    interface_id,
                    drops_count,
                    ts_high,
                    ts_low,
                    captured_length,
                    original_length,
                ) = struct.unpack_from(
                    endian + "HHIIII",
                    body,
                    0,
                )


                if interface_id >= len(
                    interfaces
                ):

                    raise RuntimeError(
                        "Packet Block references unknown interface."
                    )


                if (
                    20 + captured_length
                    >
                    len(body)
                ):

                    raise RuntimeError(
                        "Packet Block captured length exceeds block."
                    )


                frame = body[
                    20:
                    20 + captured_length
                ]


                interface = interfaces[
                    interface_id
                ]


                raw_timestamp = (
                    (
                        int(ts_high)
                        <<
                        32
                    )
                    |
                    int(ts_low)
                )


                timestamp_us = timestamp_to_us(
                    raw_timestamp,
                    interface[
                        "timestamp"
                    ],
                )


            # -------------------------------------------------------------
            # SIMPLE PACKET BLOCK
            #
            # No timestamp/interface ID -> lifecycle semantics cannot be
            # reconstructed faithfully. Fail instead of inventing values.
            # -------------------------------------------------------------

            elif block_type == PCAPNG_SPB:

                counters[
                    "unsupported_simple_packet_blocks"
                ] += 1


                raise RuntimeError(
                    "PCAPNG Simple Packet Block encountered."
                )


            else:

                continue


            # -------------------------------------------------------------
            # PACKET ACCOUNTING
            # -------------------------------------------------------------

            counters[
                "raw_packet_count"
            ] += 1


            counters[
                "captured_frame_bytes"
            ] += int(
                len(frame)
            )


            if (
                interface[
                    "linktype"
                ]
                !=
                LINKTYPE_ETHERNET
            ):

                counters[
                    "unsupported_linktype_packet_count"
                ] += 1


                parsed = None


            else:

                parsed = parse_ipv4_packet(
                    frame
                )


            if parsed is None:

                counters[
                    "non_ipv4_packet_count"
                ] += 1


            else:

                counters[
                    "valid_ipv4_packet_count"
                ] += 1


                counters[
                    "captured_ipv4_bytes"
                ] += int(
                    parsed[
                        "captured_ipv4_bytes"
                    ]
                )


                if parsed[
                    "protocol"
                ] == 6:

                    counters[
                        "parser_tcp_count"
                    ] += 1


                elif parsed[
                    "protocol"
                ] == 17:

                    counters[
                        "parser_udp_count"
                    ] += 1


                else:

                    counters[
                        "parser_other0_count"
                    ] += 1


                add_packet_to_flows(
                    active,
                    parsed,
                    timestamp_us,
                    counters,
                )


            # -------------------------------------------------------------
            # EXACT RAW PACKET PREFIX BOUNDARY
            # -------------------------------------------------------------

            if (
                packet_limit is not None
                and
                counters[
                    "raw_packet_count"
                ]
                >=
                int(
                    packet_limit
                )
            ):

                counters[
                    "pcap_file_bytes_consumed"
                ] = int(
                    f.tell()
                )

                break


        if (
            counters[
                "pcap_file_bytes_consumed"
            ]
            ==
            0
        ):

            counters[
                "pcap_file_bytes_consumed"
            ] = int(
                f.tell()
            )


    # =========================================================================
    # DECLARED CAPTURE/SAMPLE EOF
    # =========================================================================

    eof_exportable = 0

    eof_singletons = 0

    eof_retained_packets = (
        0
    )


    for flow in active.values():

        packet_count = int(
            flow[
                "packet_count"
            ]
        )


        if packet_count > 1:

            eof_exportable += 1

            eof_retained_packets += (
                packet_count
            )


        else:

            eof_singletons += 1


    counters[
        "eof_current_exportable_flows"
    ] = int(
        eof_exportable
    )


    counters[
        "eof_singleton_discarded_flows"
    ] = int(
        eof_singletons
    )


    counters[
        "retained_packet_count_eof"
    ] = int(
        eof_retained_packets
    )


    counters[
        "exportable_flow_count"
    ] = int(
        counters[
            "finished_exportable_flows"
        ]
        +
        eof_exportable
    )


    counters[
        "retained_packet_count"
    ] = int(
        counters[
            "retained_packet_count_finished"
        ]
        +
        eof_retained_packets
    )


    counters[
        "active_flow_count_at_boundary"
    ] = int(
        len(
            active
        )
    )


    return counters


# =============================================================================
# MAIN
# =============================================================================

def main():

    if len(
        sys.argv
    ) != 2:

        raise RuntimeError(
            "Expected one JSON config path."
        )


    config_path = Path(
        sys.argv[
            1
        ]
    )


    config = json.loads(
        config_path.read_text(
            encoding="utf-8"
        )
    )


    result_path = Path(
        config[
            "result_path"
        ]
    )


    pcap_path = Path(
        config[
            "pcap_path"
        ]
    )


    mode = str(
        config[
            "mode"
        ]
    )


    affinity = [
        int(x)
        for x in config[
            "affinity"
        ]
    ]


    if hasattr(
        os,
        "sched_setaffinity",
    ):

        os.sched_setaffinity(
            0,
            set(
                affinity
            ),
        )


    # -------------------------------------------------------------------------
    # FULL VALIDATION:
    # Explicitly UNTITMED.
    # No performance timer is instantiated.
    # -------------------------------------------------------------------------

    if mode == "FULL_VALIDATION_UNTIMED":

        counts = extract(
            pcap_path,
            packet_limit=None,
        )


        result = {
            "schema":
                "stage26_raw_flow_extractor_result_v1",

            "mode":
                mode,

            "status":
                "PASS",

            "timing_performed":
                False,

            "packet_limit":
                None,

            "counts":
                counts,
        }


    # -------------------------------------------------------------------------
    # FROZEN TIMED PREFIX:
    #
    # Python startup/import/config/affinity are before timer.
    # Result serialization is after timer.
    # -------------------------------------------------------------------------

    elif mode == "BENCHMARK_TIMED":

        packet_limit = int(
            config[
                "packet_limit"
            ]
        )


        start_ns = time.perf_counter_ns()


        counts = extract(
            pcap_path,
            packet_limit=packet_limit,
        )


        stop_ns = time.perf_counter_ns()


        elapsed_ns = int(
            stop_ns
            -
            start_ns
        )


        if elapsed_ns <= 0:

            raise RuntimeError(
                "Non-positive elapsed extraction time."
            )


        elapsed_seconds = (
            elapsed_ns
            /
            1_000_000_000.0
        )


        packets_per_second = (
            counts[
                "raw_packet_count"
            ]
            /
            elapsed_seconds
        )


        bytes_per_second = (
            counts[
                "captured_frame_bytes"
            ]
            /
            elapsed_seconds
        )


        result = {
            "schema":
                "stage26_raw_flow_extractor_result_v1",

            "mode":
                mode,

            "status":
                "PASS",

            "timing_performed":
                True,

            "packet_limit":
                packet_limit,

            "elapsed_ns":
                elapsed_ns,

            "elapsed_seconds":
                elapsed_seconds,

            "packets_per_second":
                packets_per_second,

            "bytes_per_second":
                bytes_per_second,

            "MiB_per_second":
                (
                    bytes_per_second
                    /
                    1024**2
                ),

            "flows_per_second":
                (
                    counts[
                        "exportable_flow_count"
                    ]
                    /
                    elapsed_seconds
                ),

            "completed_lifecycle_flows_per_second":
                (
                    counts[
                        "finished_exportable_flows"
                    ]
                    /
                    elapsed_seconds
                ),

            "container_bytes_per_second":
                (
                    counts[
                        "pcap_file_bytes_consumed"
                    ]
                    /
                    elapsed_seconds
                ),

            "captured_ipv4_bytes_per_second":
                (
                    counts[
                        "captured_ipv4_bytes"
                    ]
                    /
                    elapsed_seconds
                ),

            "counts":
                counts,
        }


    else:

        raise RuntimeError(
            f"Unknown worker mode: {mode}"
        )


    # -------------------------------------------------------------------------
    # Un-timed result metadata / fingerprint.
    # -------------------------------------------------------------------------

    result[
        "count_summary_sha256"
    ] = hashlib.sha256(
        json.dumps(
            result[
                "counts"
            ],
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
        ).encode(
            "utf-8"
        )
    ).hexdigest()


    result[
        "disk_IO_included"
    ] = True


    result[
        "serialization_included"
    ] = False


    result[
        "Python_process_startup_included"
    ] = False


    result[
        "JVM_startup_included"
    ] = False


    result[
        "parser_semantics_identifier"
    ] = (
        "STAGE20_FROZEN_IPV4_DIRECT_TRANSPORT_TCP6_UDP17_OTHER0"
    )


    result[
        "flow_id_semantics_identifier"
    ] = (
        "BASICPACKETINFO_JAVA_SIGNED_IPV4_FIRST_DIFFERING_BYTE_"
        "SWAP_IPS_AND_PORTS_TOGETHER"
    )


    result[
        "flow_lifecycle_semantics_identifier"
    ] = (
        "CICFLOWMETER_FLOWGENERATOR_"
        "EAA853DD82F08BA5288BB7F295B471DE7313F883"
    )


    result[
        "corpus_created_or_modified"
    ] = False


    result[
        "labels_accessed"
    ] = False


    result[
        "gpu_used"
    ] = False


    atomic_json(
        result_path,
        result,
    )


if __name__ == "__main__":

    main()
'''


WORKER_PATH.write_text(
    textwrap.dedent(
        worker_source
    ).lstrip(),
    encoding="utf-8",
)


# Syntax compile ONLY.
# This does not execute the worker.
compile_result = run(
    [
        sys.executable,
        "-m",
        "py_compile",
        str(
            WORKER_PATH
        ),
    ],
    check=False,
)


if compile_result.returncode != 0:

    raise RuntimeError(
        "Extraction worker syntax failure:\n"
        +
        compile_result.stdout
    )


WORKER_SHA256 = sha256_file(
    WORKER_PATH
)


print(
    "Worker syntax compile: PASS"
)

print(
    "Worker SHA256:"
)

print(
    " ",
    WORKER_SHA256
)


# =============================================================================
# 12. FREEZE COMPONENT BOUNDARY MAP
# =============================================================================

banner(
    "STAGE26-4B1 :: FREEZE COMPONENT BOUNDARY MAP"
)


component_boundary = {
    "schema":
        "stage26_component_boundary_map_v1",

    "stage":
        26,

    "frozen_before_stage26_extraction_timing":
        True,

    "parent_commit":
        EXPECTED_PARENT,

    "component_A_raw_extraction": {
        "input":
            (
                "Exact Monday raw PCAPNG source "
                f"{PCAP_SHA256}"
            ),

        "output_boundary":
            (
                "In-memory source-faithful reconstructed flow lifecycle "
                "state/counts. No corpus and no CSV are generated."
            ),

        "measurement_status":
            "PROTOCOL_FROZEN_NOT_YET_MEASURED",

        "corpus_written":
            False,
    },

    "component_B_existing_release_corpus": {
        "source":
            "EXISTING_GITHUB_RELEASE",

        "release_tag":
            RELEASE_TAG,

        "asset":
            MONDAY_CORPUS_ASSET,

        "asset_sha256":
            MONDAY_CORPUS_SHA256,

        "flow_count":
            MONDAY_CORPUS_FLOW_COUNT,

        "encoded_bytes":
            MONDAY_CORPUS_ENCODED_BYTES,

        "member_sha256":
            MONDAY_CORPUS_MEMBER_SHA256,

        "authoritative":
            True,

        "regeneration_from_pcap":
            "FORBIDDEN",

        "representation_performance_protocol_status":
            "NOT_YET_FROZEN",

        "important_boundary":
            (
                "Corpus already lies downstream of raw parsing, flow "
                "reconstruction, supervised matching/filtering, packet "
                "selection/truncation, and masking/encoding."
            ),
    },

    "component_C_model_inference": {
        "status":
            "ALREADY_MEASURED_STAGE26_2",

        "note":
            (
                "Stage26-2 isolated model inference and excluded input "
                "generation/preprocessing from the timed interval."
            ),
    },

    "additivity_rule": {
        "A_plus_B_plus_C_is_currently_complete_end_to_end":
            False,

        "reason":
            (
                "The Release corpus is already post several preprocessing "
                "operations. A future representation/end-to-end protocol "
                "must explicitly define and measure remaining boundaries "
                "before additive end-to-end claims are allowed."
            ),
    },
}


atomic_json(
    BOUNDARY_PATH,
    component_boundary,
)


BOUNDARY_SHA256 = sha256_file(
    BOUNDARY_PATH
)


print(
    "Boundary-map SHA256:"
)

print(
    " ",
    BOUNDARY_SHA256
)


# =============================================================================
# 13. FREEZE EXTRACTION SUBPROTOCOL
# =============================================================================

banner(
    "STAGE26-4B1 :: FREEZE EXTRACTION SUBPROTOCOL"
)


extraction_protocol = {
    "schema":
        "stage26_extraction_protocol_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-4B1",

    "status":
        "FROZEN_BEFORE_FIRST_EXTRACTION_MEASUREMENT",

    "frozen_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_PARENT,

    "upstream": {
        "measurement_protocol_sha256":
            EXPECTED_MEASUREMENT_PROTOCOL_SHA256,

        "stage20_monday_geometry_sha256":
            EXPECTED_GEOMETRY_SHA256,

        "source_restoration_receipt_sha256":
            EXPECTED_SOURCE_RECEIPT_SHA256,

        "extractor_inventory_sha256":
            EXPECTED_EXTRACTOR_INVENTORY_SHA256,

        "release_corpus_structure_audit_sha256":
            EXPECTED_CORPUS_STRUCTURE_SHA256,

        "release_corpus_geometry_audit_sha256":
            EXPECTED_CORPUS_GEOMETRY_SHA256,

        "component_boundary_map_sha256":
            BOUNDARY_SHA256,
    },

    "source": {
        "filename":
            PCAP_FILENAME,

        "container":
            PCAP_CONTAINER,

        "canonical_path":
            str(
                MONDAY_PCAP
            ),

        "resolved_path_at_freeze":
            str(
                resolved_pcap
            ),

        "size_bytes":
            PCAP_SIZE_BYTES,

        "sha256":
            PCAP_SHA256,

        "historical_raw_packet_count":
            RAW_PACKET_COUNT,

        "source_selection":
            "MONDAY_STAGE20_TRAIN_SOURCE",

        "source_selection_used_stage26_performance_results":
            False,
    },

    "extractor": {
        "implementation_id":
            "STAGE26_SOURCE_FAITHFUL_RAW_FLOW_EXTRACTOR_V1",

        "worker_path_at_freeze":
            str(
                WORKER_PATH
            ),

        "worker_sha256":
            WORKER_SHA256,

        "python_version":
            sys.version,

        "parser_semantics_identifier":
            PARSER_SEMANTICS_ID,

        "flow_id_semantics_identifier":
            FLOW_ID_SEMANTICS_ID,

        "flow_lifecycle_semantics_identifier":
            FLOW_LIFECYCLE_ID,

        "flow_timeout_us":
            FLOW_TIMEOUT_US,

        "lifecycle_reference_repository":
            "ahlashkari/CICFlowMeter",

        "lifecycle_reference_commit":
            CICFLOWMETER_COMMIT,

        "lifecycle_reference_tree":
            CICFLOWMETER_TREE,

        "java_packetreader_used":
            False,

        "reason":
            (
                "Stage20 separately freezes custom parser semantics and "
                "CICFlowMeter lifecycle semantics. The current runtime "
                "also lacks the original jnetpcap/libpcap execution path."
            ),
    },

    "full_untimed_correctness_gate": {
        "required_before_any_timed_extraction":
            True,

        "timing_performed":
            False,

        "scope":
            "ENTIRE_EXACT_MONDAY_PCAP",

        "expected_exact_counts": {
            "raw_packet_count":
                RAW_PACKET_COUNT,

            "valid_ipv4_packet_count":
                VALID_IPV4_PACKET_COUNT,

            "non_ipv4_packet_count":
                NON_IPV4_PACKET_COUNT,

            "parser_tcp_count":
                PARSER_TCP_COUNT,

            "parser_udp_count":
                PARSER_UDP_COUNT,

            "parser_other0_count":
                PARSER_OTHER0_COUNT,

            "exportable_flow_count":
                RAW_EXPORTABLE_FLOW_COUNT,

            "retained_packet_count":
                RETAINED_PACKET_COUNT,

            "max_active_flows":
                MAX_ACTIVE_FLOWS,

            "fin_exportable_flows":
                FIN_FLOW_COUNT,

            "timeout_exportable_flows":
                TIMEOUT_FLOW_COUNT,

            "eof_current_exportable_flows":
                EOF_CURRENT_FLOW_COUNT,

            "discarded_singleton_timeout_flows":
                TIMEOUT_SINGLETON_DISCARDED,

            "eof_singleton_discarded_flows":
                EOF_SINGLETON_DISCARDED,
        },

        "failure_rule":
            (
                "Any mismatch blocks extraction timing. Only a narrowly "
                "scoped parser/lifecycle implementation correction that "
                "restores the already-frozen Stage20 semantics is allowed. "
                "The source, timed packet bounds, repetitions, metrics, "
                "hardware condition, and failure rules may not change."
            ),
    },

    "timed_sample": {
        "sampling_rule":
            "DETERMINISTIC_CONTIGUOUS_RAW_PACKET_PREFIX",

        "raw_packet_start_index":
            TIMED_RAW_PACKET_START,

        "raw_packet_end_index":
            TIMED_RAW_PACKET_END,

        "raw_packet_count":
            TIMED_RAW_PACKET_COUNT,

        "fraction_of_full_raw_packet_population":
            TIMED_SAMPLE_FRACTION,

        "reason":
            (
                "Bounded reproducible sequential-source throughput sample "
                "frozen before any extraction performance measurement; "
                "independent of Stage26 model latency/memory results."
            ),

        "sample_boundary":
            (
                "Packet 1,000,000 is treated as declared capture EOF for "
                "flow-count throughput accounting."
            ),
    },

    "execution": {
        "cpu_mode":
            CPU_MODE,

        "affinity":
            CPU_AFFINITY,

        "thread_count":
            CPU_THREAD_COUNT,

        "fresh_process_per_repetition":
            True,

        "repetitions":
            TIMED_REPETITIONS,

        "python_hash_seed":
            0,

        "timeout_seconds_per_repetition":
            RESOURCE_TIMEOUT_SECONDS,

        "gpu":
            False,

        "environment_gate": {
            "cpu_utilization_percent_max":
                CPU_UTILIZATION_MAX_PERCENT,

            "cpu_utilization_sampling_seconds":
                CPU_UTILIZATION_SAMPLE_SECONDS,

            "available_ram_gib_min":
                AVAILABLE_RAM_MIN_GIB,

            "attempts":
                ENV_GATE_RETRIES,

            "retry_cooldown_seconds":
                ENV_GATE_RETRY_COOLDOWN_SECONDS,
        },
    },

    "measurement_boundary": {
        "timer":
            "time.perf_counter_ns",

        "timer_start":
            (
                "Immediately before opening/reading the PCAP and beginning "
                "bounded raw extraction."
            ),

        "timer_stop":
            (
                "Immediately after in-memory flow lifecycle reconstruction "
                "and declared sample-EOF flow accounting."
            ),

        "disk_IO_included_boolean":
            True,

        "serialization_included_boolean":
            False,

        "JVM_startup_included_boolean":
            False,

        "Python_process_startup_included_boolean":
            False,

        "result_json_serialization":
            "AFTER_TIMER",

        "os_page_cache_policy":
            (
                "No privileged cache flushing/manipulation. Application "
                "file-read cost is inside the timed boundary; all five raw "
                "repetitions are retained."
            ),
    },

    "required_metrics": {
        "packets_per_second":
            (
                "raw packet records processed / elapsed seconds"
            ),

        "bytes_per_second":
            (
                "sum of captured frame bytes processed / elapsed seconds"
            ),

        "MiB_per_second":
            (
                "bytes_per_second / 2^20"
            ),

        "flows_per_second":
            (
                "exportable flows including declared sample-EOF exports "
                "/ elapsed seconds"
            ),
    },

    "additional_descriptive_metrics": [
        "completed_lifecycle_flows_per_second",
        "container_bytes_per_second",
        "captured_ipv4_bytes_per_second",
        "valid_ipv4_packet_count",
        "parser_tcp_count",
        "parser_udp_count",
        "parser_other0_count",
        "fin_exportable_flows",
        "timeout_exportable_flows",
        "eof_current_exportable_flows",
        "max_active_flows",
    ],

    "summary_policy": {
        "raw_repetitions_retained":
            True,

        "repetition_count":
            TIMED_REPETITIONS,

        "summary_statistics":
            [
                "median",
                "minimum",
                "maximum",
            ],

        "p95_p99_not_inferred_from_five_repetitions":
            True,
    },

    "failure_policy": {
        "timeout":
            (
                "TIMEOUT_RESOURCE_LIMIT after 600 seconds; no timeout "
                "or packet-bound adaptation."
            ),

        "oom":
            (
                "RESOURCE_LIMIT_OOM; no post-hoc sample reduction."
            ),

        "implementation_mismatch":
            (
                "Block timing and perform only source-faithful correctness "
                "diagnosis against already-frozen Stage20 anchors."
            ),

        "invalid_environment":
            (
                "INVALID_ENVIRONMENT if precondition gate fails all "
                "attempts."
            ),
    },

    "corpus_policy": {
        "recreate_stage20_release_corpus_from_pcap":
            False,

        "write_new_packet_corpus":
            False,

        "write_new_flow_corpus":
            False,

        "write_cicflowmeter_csv":
            False,

        "existing_authoritative_release_tag":
            RELEASE_TAG,

        "existing_authoritative_monday_asset":
            MONDAY_CORPUS_ASSET,

        "existing_authoritative_monday_asset_sha256":
            MONDAY_CORPUS_SHA256,

        "existing_release_flow_count":
            MONDAY_CORPUS_FLOW_COUNT,

        "raw_exportable_flow_count":
            RAW_EXPORTABLE_FLOW_COUNT,

        "counts_are_expected_to_differ":
            True,
    },

    "claim_boundary": {
        "measured":
            (
                "Sequential raw PCAPNG read, Ethernet/VLAN IPv4 parsing, "
                "direct TCP/UDP/OTHER0 normalization, source-faithful "
                "bidirectional flow keying and CICFlowMeter-style "
                "timeout/FIN/EOF lifecycle reconstruction."
            ),

        "not_measured":
            [
                "Stage20 supervised flow-to-label matching/filtering",
                "packet truncation/selection to maximum 64 packets",
                "IPv4/TCP/UDP masking",
                "compact corpus creation",
                "compact corpus serialization",
                "dense 64x256 representation materialization",
                "float32 /255 model scaling",
                "70-feature CICFlowMeter CSV feature-vector serialization",
                "model inference",
                "GPU execution",
            ],

        "complete_end_to_end_claim_allowed":
            False,
    },

    "scientific_rules": {
        "protocol_frozen_before_first_extraction_measurement":
            True,

        "full_correctness_gate_before_first_timing":
            True,

        "inference_results_used_to_choose_source":
            False,

        "post_result_packet_bound_change":
            False,

        "corpus_regeneration_from_pcap":
            False,

        "labels_accessed":
            False,

        "Thursday_accessed":
            False,

        "Friday_accessed":
            False,

        "gpu_used":
            False,
    },

    "next_actions": [
        (
            "Git-anchor and remotely verify this extraction protocol "
            "and exact worker."
        ),
        (
            "Run one full-Monday UNTIMED correctness validation."
        ),
        (
            "Only if every historical count matches exactly, execute "
            "the five frozen timed 1,000,000-packet repetitions."
        ),
        (
            "Freeze any Release-corpus representation/materialization "
            "performance protocol separately before measuring it."
        ),
    ],
}


atomic_json(
    PROTOCOL_PATH,
    extraction_protocol,
)


PROTOCOL_SHA256 = sha256_file(
    PROTOCOL_PATH
)


print(
    "Extraction protocol SHA256:"
)

print(
    " ",
    PROTOCOL_SHA256
)


print(
    "\nFrozen timed sample:"
)

print(
    "  packets:",
    f"{TIMED_RAW_PACKET_START:,}"
    " .. "
    f"{TIMED_RAW_PACKET_END:,}"
)

print(
    "  count  :",
    f"{TIMED_RAW_PACKET_COUNT:,}"
)

print(
    "  fraction of Monday:",
    f"{TIMED_SAMPLE_FRACTION:.6%}"
)

print(
    "  repetitions:",
    TIMED_REPETITIONS
)


# =============================================================================
# 14. FREEZE RECEIPT
# =============================================================================

freeze_receipt = {
    "schema":
        "stage26_4b1_extraction_protocol_freeze_receipt_v1",

    "stage":
        26,

    "checkpoint":
        "STAGE26-4B1",

    "status":
        "PASS_PROTOCOL_FROZEN_BEFORE_EXTRACTION",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_PARENT,

    "measurement_protocol_sha256":
        EXPECTED_MEASUREMENT_PROTOCOL_SHA256,

    "stage20_geometry_sha256":
        EXPECTED_GEOMETRY_SHA256,

    "source_restoration_receipt_sha256":
        EXPECTED_SOURCE_RECEIPT_SHA256,

    "extractor_inventory_sha256":
        EXPECTED_EXTRACTOR_INVENTORY_SHA256,

    "release_corpus_geometry_audit_sha256":
        EXPECTED_CORPUS_GEOMETRY_SHA256,

    "component_boundary_map_sha256":
        BOUNDARY_SHA256,

    "worker_sha256":
        WORKER_SHA256,

    "extraction_protocol_sha256":
        PROTOCOL_SHA256,

    "full_untimed_validation_required":
        True,

    "timed_raw_packet_count":
        TIMED_RAW_PACKET_COUNT,

    "timed_repetitions":
        TIMED_REPETITIONS,

    "disk_IO_included":
        True,

    "serialization_included":
        False,

    "Python_process_startup_included":
        False,

    "JVM_startup_included":
        False,

    "existing_release_corpus_authoritative":
        True,

    "corpus_regeneration_from_pcap":
        False,

    "pcap_opened_or_iterated_by_freeze_cell":
        False,

    "extraction_performed":
        False,

    "timing_performed":
        False,

    "models_loaded":
        False,

    "labels_accessed":
        False,

    "gpu_used":
        False,

    "next_action":
        "GIT_ANCHOR_BEFORE_FULL_UNTIMED_VALIDATION",
}


atomic_json(
    FREEZE_RECEIPT,
    freeze_receipt,
)


FREEZE_RECEIPT_SHA256 = sha256_file(
    FREEZE_RECEIPT
)


print(
    "Freeze receipt SHA256:"
)

print(
    " ",
    FREEZE_RECEIPT_SHA256
)


# =============================================================================
# 15. BUILD DURABLE REPOSITORY LOCK PACKAGE
# =============================================================================

banner(
    "STAGE26-4B1 :: BUILD REPOSITORY LOCK PACKAGE"
)


if REPO_LOCK_DIR.exists():

    raise RuntimeError(
        "Stage26-4B extraction lock directory already exists; "
        "stop rather than overwrite a prior freeze."
    )


REPO_LOCK_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


files_to_copy = [
    WORKER_PATH,
    PROTOCOL_PATH,
    BOUNDARY_PATH,
    FREEZE_RECEIPT,
]


for source in files_to_copy:

    shutil.copy2(
        source,
        REPO_LOCK_DIR
        / source.name,
    )


UPSTREAM_REFERENCE = (
    REPO_LOCK_DIR
    / "stage26_4b_upstream_provenance.json"
)


atomic_json(
    UPSTREAM_REFERENCE,
    {
        "schema":
            "stage26_4b_upstream_provenance_v1",

        "parent_commit":
            EXPECTED_PARENT,

        "measurement_protocol_sha256":
            EXPECTED_MEASUREMENT_PROTOCOL_SHA256,

        "stage20_monday_geometry": {
            "repo_relative_path":
                str(
                    GEOMETRY_PROFILE.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_GEOMETRY_SHA256,
        },

        "raw_source": {
            "filename":
                PCAP_FILENAME,

            "size_bytes":
                PCAP_SIZE_BYTES,

            "sha256":
                PCAP_SHA256,

            "container":
                PCAP_CONTAINER,
        },

        "historical_raw_counts": {
            "raw_packets":
                RAW_PACKET_COUNT,

            "valid_ipv4":
                VALID_IPV4_PACKET_COUNT,

            "exportable_flows":
                RAW_EXPORTABLE_FLOW_COUNT,

            "retained_packets":
                RETAINED_PACKET_COUNT,
        },

        "existing_release_corpus": {
            "tag":
                RELEASE_TAG,

            "asset":
                MONDAY_CORPUS_ASSET,

            "size_bytes":
                MONDAY_CORPUS_SIZE,

            "sha256":
                MONDAY_CORPUS_SHA256,

            "flow_count":
                MONDAY_CORPUS_FLOW_COUNT,

            "geometry_audit_sha256":
                EXPECTED_CORPUS_GEOMETRY_SHA256,

            "authoritative":
                True,

            "regeneration_from_pcap":
                False,
        },

        "extractor_semantics": {
            "parser":
                PARSER_SEMANTICS_ID,

            "flow_id":
                FLOW_ID_SEMANTICS_ID,

            "lifecycle":
                FLOW_LIFECYCLE_ID,

            "cicflowmeter_commit":
                CICFLOWMETER_COMMIT,
        },
    },
)


# =============================================================================
# 16. LOCK PACKAGE MANIFEST
# =============================================================================

PACKAGE_MANIFEST = (
    REPO_LOCK_DIR
    / "stage26_4b_extraction_lock_manifest.json"
)


manifest_rows = []


for path in sorted(
    REPO_LOCK_DIR.iterdir()
):

    if (
        path.is_file()
        and
        path != PACKAGE_MANIFEST
    ):

        manifest_rows.append(
            {
                "repo_relative_path":
                    str(
                        path.relative_to(
                            REPO
                        )
                    ),

                "size_bytes":
                    int(
                        path.stat().st_size
                    ),

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )


atomic_json(
    PACKAGE_MANIFEST,
    {
        "schema":
            "stage26_4b_extraction_lock_manifest_v1",

        "stage":
            26,

        "checkpoint":
            "STAGE26-4B1",

        "status":
            "READY_FOR_GIT_ANCHOR",

        "parent_commit":
            EXPECTED_PARENT,

        "worker_sha256":
            WORKER_SHA256,

        "extraction_protocol_sha256":
            PROTOCOL_SHA256,

        "component_boundary_map_sha256":
            BOUNDARY_SHA256,

        "freeze_receipt_sha256":
            FREEZE_RECEIPT_SHA256,

        "file_count_excluding_manifest":
            len(
                manifest_rows
            ),

        "files":
            manifest_rows,

        "scientific_state": {
            "protocol_frozen":
                True,

            "worker_frozen":
                True,

            "existing_release_corpus_authoritative":
                True,

            "corpus_regenerated":
                False,

            "pcap_opened":
                False,

            "pcap_packets_iterated":
                0,

            "full_validation_performed":
                False,

            "extraction_timing_performed":
                False,

            "gpu_used":
                False,
        },
    },
)


PACKAGE_MANIFEST_SHA256 = sha256_file(
    PACKAGE_MANIFEST
)


print(
    "Lock manifest SHA256:"
)

print(
    " ",
    PACKAGE_MANIFEST_SHA256
)


# =============================================================================
# 17. STATIC / REPOSITORY AUDIT
# =============================================================================

banner(
    "STAGE26-4B1 :: STATIC AUDIT"
)


# Ensure worker was only syntax-compiled, never invoked.
if not WORKER_PATH.exists():

    raise RuntimeError(
        "Worker missing."
    )


# Source existence only — no file opening.
if int(
    resolved_pcap.stat().st_size
) != PCAP_SIZE_BYTES:

    raise RuntimeError(
        "Raw source size changed during protocol freeze."
    )


# Mandatory original Stage26 fields.
measurement_boundary = extraction_protocol[
    "measurement_boundary"
]


for key in [
    "disk_IO_included_boolean",
    "serialization_included_boolean",
    "JVM_startup_included_boolean",
    "Python_process_startup_included_boolean",
]:

    if key not in measurement_boundary:

        raise RuntimeError(
            f"Missing mandatory extraction boundary: {key}"
        )


for metric in [
    "packets_per_second",
    "bytes_per_second",
    "MiB_per_second",
    "flows_per_second",
]:

    if (
        metric
        not in extraction_protocol[
            "required_metrics"
        ]
    ):

        raise RuntimeError(
            f"Missing required extraction metric: {metric}"
        )


repo_status_after = git(
    "status",
    "--porcelain",
)


print(
    "Repository status after freeze:"
)

print(
    repo_status_after
    if repo_status_after
    else
    "<clean>"
)


if not repo_status_after:

    raise RuntimeError(
        "Expected uncommitted Stage26-4B lock package."
    )


unexpected = []


for line in repo_status_after.splitlines():

    rel = line[
        3:
    ]


    if not rel.startswith(
        "results/stage26_deployment_profiling/"
        "stage26_4b_extraction_protocol_lock/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected Git modifications outside Stage26-4B lock:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 18. CLOSURE
# =============================================================================

banner(
    "STAGE26-4B1 RAW EXTRACTION FREEZE COMPLETE"
)


print(
    "PARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nRAW SOURCE:"
)

print(
    " ",
    PCAP_FILENAME
)

print(
    "  bytes :",
    PCAP_SIZE_BYTES
)

print(
    "  SHA256:",
    PCAP_SHA256
)


print(
    "\nRAW HISTORICAL POPULATION:"
)

print(
    "  packets           :",
    f"{RAW_PACKET_COUNT:,}"
)

print(
    "  valid IPv4        :",
    f"{VALID_IPV4_PACKET_COUNT:,}"
)

print(
    "  exportable flows  :",
    f"{RAW_EXPORTABLE_FLOW_COUNT:,}"
)

print(
    "  retained packets  :",
    f"{RETAINED_PACKET_COUNT:,}"
)


print(
    "\nAUTHORITATIVE EXISTING RELEASE CORPUS:"
)

print(
    "  tag               :",
    RELEASE_TAG
)

print(
    "  asset             :",
    MONDAY_CORPUS_ASSET
)

print(
    "  flows             :",
    f"{MONDAY_CORPUS_FLOW_COUNT:,}"
)

print(
    "  corpus regenerated:",
    "NO"
)


print(
    "\nEXTRACTION WORKER SHA256:"
)

print(
    " ",
    WORKER_SHA256
)


print(
    "\nEXTRACTION PROTOCOL SHA256:"
)

print(
    " ",
    PROTOCOL_SHA256
)


print(
    "\nCOMPONENT BOUNDARY SHA256:"
)

print(
    " ",
    BOUNDARY_SHA256
)


print(
    "\nFREEZE RECEIPT SHA256:"
)

print(
    " ",
    FREEZE_RECEIPT_SHA256
)


print(
    "\nLOCK MANIFEST SHA256:"
)

print(
    " ",
    PACKAGE_MANIFEST_SHA256
)


print(
    "\nTIMED SAMPLE FROZEN:"
)

print(
    "  raw packets       : 1 .. 1,000,000"
)

print(
    "  fraction Monday   :",
    f"{TIMED_SAMPLE_FRACTION:.6%}"
)

print(
    "  CPU               : 1 physical core"
)

print(
    "  repetitions       :",
    TIMED_REPETITIONS
)

print(
    "  disk I/O included : YES"
)

print(
    "  serialization     : NO"
)

print(
    "  Python startup    : NO"
)

print(
    "  JVM startup       : NO"
)


print(
    "\nCORRECTNESS GATE:"
)

print(
    "  Full 11,709,971-packet Monday pass must exactly reproduce"
)

print(
    "  ALL frozen parser/lifecycle counts BEFORE any timing."
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  RAW EXTRACTION PROTOCOL FROZEN      : YES"
)

print(
    "  RAW EXTRACTION WORKER FROZEN        : YES"
)

print(
    "  RELEASE CORPUS AUTHORITATIVE        : YES"
)

print(
    "  CORPUS RECREATED FROM PCAP          : NO"
)

print(
    "  PCAP OPENED / ITERATED              : NO"
)

print(
    "  EXTRACTION PERFORMED                : NO"
)

print(
    "  EXTRACTION TIMING                   : NO"
)

print(
    "  REPRESENTATION TIMING               : NO"
)

print(
    "  COMPLETE E2E ADDITIVITY CLAIM       : NO"
)

print(
    "  GPU                                 : NO"
)


print(
    "\nNEXT:"
)

print(
    "  STAGE26-4B1-GIT — commit, push and remotely verify this"
)

print(
    "  extraction protocol lock BEFORE the first full-Monday"
)

print(
    "  untimed correctness validation."
)


STAGE26-4B1 :: SCIENTIFIC ANCHOR
Expected parent: 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Local HEAD     : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
origin/main    : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Repo clean     : True

STAGE26-4B1 :: UPSTREAM PROVENANCE GATE
Stage26 measurement protocol          PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
Stage20 Monday geometry               PASS 3a26d6499334c12ea4e9272aef4250761c6cf4399e7fb4d33ef236f12d0b7272
Stage26-4A source receipt             PASS a7d25f504c5a1d75f6cb46b94bd498ae6ca236a3872be5f868ce9160ec269bb3
Stage26-4B0 extractor inventory       PASS 8e8188f2606dfbfdc7d662a378df29b17c0540416d061417d288b02ff7a0f686
Monday corpus structure audit         PASS a0ed17fa8d26049f53766e32981dff72724d79fbd573e7aab79c926a36833c3b
Monday corpus geometry audit          PASS 47ad682502cfe3ca98d794e9a800263fed1b1c8b5331f5400160d27a518d129a

STAGE26-4B1 :: ORIGINAL EXTRACTION GATE
{
  "inference_results_may_select_pcap":

In [13]:
# =============================================================================
# STAGE26-4B1-GIT
# COMMIT + PUSH + REMOTELY VERIFY EXTRACTION PROTOCOL LOCK
#
# ABSOLUTE RULES:
#   - NO PCAP OPEN / HASH / ITERATION
#   - NO EXTRACTION
#   - NO TIMING
#   - NO MODELS
#   - NO HOLDOUT
#   - NO GPU
#
# This cell ONLY:
#   1. verifies the frozen Stage26-4B1 package,
#   2. commits it,
#   3. pushes main,
#   4. fetches origin/main,
#   5. verifies every committed file byte-for-byte from the remote-tracking tree.
# =============================================================================

from __future__ import annotations

import os
import json
import hashlib
import stat
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. FROZEN EXPECTATIONS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

LOCK_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_4b_extraction_protocol_lock"
)

LOCK_DIR = (
    REPO
    / LOCK_REL
)

EXPECTED_PARENT = (
    "347d93f21d454cc5bda2c45889c890c67cdf0ecc"
)

EXPECTED_WORKER_SHA256 = (
    "ba34c5ab1a8e0bdb22acf5d6be3430cc2b31fdec04dc333dcf3ef7344abe44e6"
)

EXPECTED_PROTOCOL_SHA256 = (
    "12f7f8f0332d92694a827e3473dc1cef66939cf1cf1c49c17ad1699e60d929b0"
)

EXPECTED_BOUNDARY_SHA256 = (
    "3f798bf9665dabe245e627b3a21ae0ecd2f837f84b091ad07bc13f60f97458cf"
)

EXPECTED_FREEZE_RECEIPT_SHA256 = (
    "3152b3f35a8cf7920ce6d60f22de49f1f6fdc7bd36c1c35d763d10015f09f811"
)

EXPECTED_MANIFEST_SHA256 = (
    "7fe04d2628e1bfe7da5551192f45821b5406b3b3087976c4334bcd74224fc2d3"
)

COMMIT_SUBJECT = (
    "stage26: freeze extraction profiling protocol"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 118
    )

    print(text)

    print(
        "=" * 118
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            + " ".join(
                map(
                    str,
                    cmd,
                )
            )
            + "\n\n"
            + output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        check=True,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:

                break

            h.update(
                chunk
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        check=True,
        text=False,
    )

    return bytes(
        p.stdout
    )


# =============================================================================
# 2. PRE-COMMIT SCIENTIFIC GATE
# =============================================================================

banner(
    "STAGE26-4B1-GIT :: PRE-COMMIT GATE"
)


head_before = git(
    "rev-parse",
    "HEAD",
)

remote_before = git(
    "rev-parse",
    "origin/main",
)

status_before = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head_before
)

print(
    "origin/main    :",
    remote_before
)

print(
    "\nGit status:"
)

print(
    status_before
    if status_before
    else "<clean>"
)


if head_before != EXPECTED_PARENT:

    raise RuntimeError(
        "Local HEAD changed before Stage26-4B1 commit."
    )


if remote_before != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Stage26-4B1 commit."
    )


if not status_before:

    raise RuntimeError(
        "Expected the uncommitted Stage26-4B1 lock package."
    )


unexpected = []


for line in status_before.splitlines():

    rel = line[
        3:
    ]

    if not rel.startswith(
        str(
            LOCK_REL
        )
        +
        "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository modifications:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 3. VERIFY LOCAL LOCK PACKAGE HASHES
# =============================================================================

banner(
    "STAGE26-4B1-GIT :: LOCAL LOCK HASH GATE"
)


expected_named_hashes = {
    "stage26_raw_flow_extractor_v1.py":
        EXPECTED_WORKER_SHA256,

    "stage26_extraction_protocol.json":
        EXPECTED_PROTOCOL_SHA256,

    "stage26_component_boundary_map.json":
        EXPECTED_BOUNDARY_SHA256,

    "stage26_4b1_extraction_protocol_freeze_receipt.json":
        EXPECTED_FREEZE_RECEIPT_SHA256,

    "stage26_4b_extraction_lock_manifest.json":
        EXPECTED_MANIFEST_SHA256,
}


for filename, expected_sha in expected_named_hashes.items():

    path = (
        LOCK_DIR
        / filename
    )

    if not path.exists():

        raise FileNotFoundError(
            path
        )

    actual_sha = sha256_file(
        path
    )

    passed = (
        actual_sha
        ==
        expected_sha
    )

    print(
        f"{filename:58s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_sha}"
    )

    if not passed:

        raise RuntimeError(
            f"Local frozen hash mismatch: {filename}"
        )


manifest_path = (
    LOCK_DIR
    / "stage26_4b_extraction_lock_manifest.json"
)

manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)


if manifest[
    "status"
] != "READY_FOR_GIT_ANCHOR":

    raise RuntimeError(
        "Unexpected lock manifest state."
    )


if manifest[
    "parent_commit"
] != EXPECTED_PARENT:

    raise RuntimeError(
        "Lock manifest parent mismatch."
    )


if manifest[
    "worker_sha256"
] != EXPECTED_WORKER_SHA256:

    raise RuntimeError(
        "Worker hash mismatch inside manifest."
    )


if manifest[
    "extraction_protocol_sha256"
] != EXPECTED_PROTOCOL_SHA256:

    raise RuntimeError(
        "Protocol hash mismatch inside manifest."
    )


if manifest[
    "component_boundary_map_sha256"
] != EXPECTED_BOUNDARY_SHA256:

    raise RuntimeError(
        "Boundary hash mismatch inside manifest."
    )


if manifest[
    "freeze_receipt_sha256"
] != EXPECTED_FREEZE_RECEIPT_SHA256:

    raise RuntimeError(
        "Freeze receipt hash mismatch inside manifest."
    )


# Validate every file referenced by the manifest.
for row in manifest[
    "files"
]:

    rel = Path(
        row[
            "repo_relative_path"
        ]
    )

    path = (
        REPO
        / rel
    )

    if not path.exists():

        raise FileNotFoundError(
            path
        )

    actual_size = int(
        path.stat().st_size
    )

    actual_sha = sha256_file(
        path
    )

    if actual_size != int(
        row[
            "size_bytes"
        ]
    ):

        raise RuntimeError(
            f"Manifest size mismatch: {rel}"
        )

    if actual_sha != row[
        "sha256"
    ]:

        raise RuntimeError(
            f"Manifest SHA mismatch: {rel}"
        )


print(
    "\nManifest file-level validation: PASS"
)

print(
    "Files validated:",
    len(
        manifest[
            "files"
        ]
    )
)


# =============================================================================
# 4. STAGE + COMMIT
# =============================================================================

banner(
    "STAGE26-4B1-GIT :: COMMIT"
)


git(
    "add",
    str(
        LOCK_REL
    ),
)


staged = git(
    "diff",
    "--cached",
    "--name-status",
)


print(
    staged
)


if not staged:

    raise RuntimeError(
        "Nothing staged for Stage26-4B1."
    )


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit SHA :",
    commit_sha
)

print(
    "Parent     :",
    commit_parent
)

print(
    "Subject    :",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-4B1 commit has unexpected parent."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Stage26-4B1 commit subject mismatch."
    )


status_after_commit = git(
    "status",
    "--porcelain",
)


print(
    "Repo clean after commit:",
    status_after_commit == ""
)


if status_after_commit:

    raise RuntimeError(
        "Repository not clean immediately after commit."
    )


# =============================================================================
# 5. GET GITHUB TOKEN SAFELY FROM KAGGLE SECRET
# =============================================================================

banner(
    "STAGE26-4B1-GIT :: AUTHENTICATED PUSH"
)


try:

    from kaggle_secrets import UserSecretsClient

except Exception as exc:

    raise RuntimeError(
        "Kaggle UserSecretsClient unavailable."
    ) from exc


secret_client = UserSecretsClient()


github_token = secret_client.get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle secret GITHUB_TOKEN is empty or unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token length :",
    len(
        github_token
    )
)

print(
    "Token value  : <not printed>"
)


# Temporary askpass script.
with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )

    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )

    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    push_result = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
        check=True,
        text=True,
    )


    print(
        push_result.stdout.strip()
    )


# Explicitly discard in-memory reference after push.
github_token = None


# =============================================================================
# 6. FETCH + REMOTE COMMIT IDENTITY VERIFICATION
# =============================================================================

banner(
    "STAGE26-4B1-GIT :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Subject    :",
    remote_subject
)

print(
    "Parent     :",
    remote_parent
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed unexpectedly after push."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "Remote main does not equal the Stage26-4B1 commit."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote commit subject mismatch."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote Stage26-4B1 parent mismatch."
    )


# =============================================================================
# 7. BYTE-VERIFY EVERY REMOTE-TRACKING FILE
# =============================================================================

banner(
    "STAGE26-4B1-GIT :: REMOTE BYTE VERIFICATION"
)


# First retrieve the manifest exactly as committed on origin/main.
remote_manifest_rel = (
    LOCK_REL
    / "stage26_4b_extraction_lock_manifest.json"
)

remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    str(
        remote_manifest_rel
    ),
)

remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "Remote manifest SHA256:"
)

print(
    " ",
    remote_manifest_sha
)


if remote_manifest_sha != EXPECTED_MANIFEST_SHA256:

    raise RuntimeError(
        "Remote manifest byte hash mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


remote_rows = list(
    remote_manifest[
        "files"
    ]
)


for row in remote_rows:

    rel = row[
        "repo_relative_path"
    ]

    remote_bytes = git_blob_bytes(
        "origin/main",
        rel,
    )

    actual_size = len(
        remote_bytes
    )

    actual_sha = sha256_bytes(
        remote_bytes
    )

    expected_size = int(
        row[
            "size_bytes"
        ]
    )

    expected_sha = row[
        "sha256"
    ]

    passed = (
        actual_size == expected_size
        and
        actual_sha == expected_sha
    )

    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{rel}"
    )

    if not passed:

        raise RuntimeError(
            f"Remote byte verification failed: {rel}"
        )


# Also verify the manifest itself against the frozen Cell-20 hash.
print(
    "\nPASS "
    f"{len(remote_manifest_bytes):10,d} B "
    f"{remote_manifest_sha} "
    f"{remote_manifest_rel}"
)


# =============================================================================
# 8. SCIENTIFIC CONTENT VERIFICATION FROM REMOTE TREE
# =============================================================================

banner(
    "STAGE26-4B1-GIT :: REMOTE SCIENTIFIC CONTENT AUDIT"
)


protocol_rel = (
    LOCK_REL
    / "stage26_extraction_protocol.json"
)

protocol_remote = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            protocol_rel
        ),
    ).decode(
        "utf-8"
    )
)


checks = {
    "protocol_state":
        (
            protocol_remote[
                "status"
            ]
            ==
            "FROZEN_BEFORE_FIRST_EXTRACTION_MEASUREMENT"
        ),

    "full_validation_required":
        (
            protocol_remote[
                "full_untimed_correctness_gate"
            ][
                "required_before_any_timed_extraction"
            ]
            is True
        ),

    "raw_packet_population":
        (
            protocol_remote[
                "source"
            ][
                "historical_raw_packet_count"
            ]
            ==
            11_709_971
        ),

    "raw_exportable_flow_anchor":
        (
            protocol_remote[
                "full_untimed_correctness_gate"
            ][
                "expected_exact_counts"
            ][
                "exportable_flow_count"
            ]
            ==
            529_601
        ),

    "timed_prefix":
        (
            protocol_remote[
                "timed_sample"
            ][
                "raw_packet_count"
            ]
            ==
            1_000_000
        ),

    "timed_repetitions":
        (
            protocol_remote[
                "execution"
            ][
                "repetitions"
            ]
            ==
            5
        ),

    "disk_io_included":
        (
            protocol_remote[
                "measurement_boundary"
            ][
                "disk_IO_included_boolean"
            ]
            is True
        ),

    "serialization_excluded":
        (
            protocol_remote[
                "measurement_boundary"
            ][
                "serialization_included_boolean"
            ]
            is False
        ),

    "python_startup_excluded":
        (
            protocol_remote[
                "measurement_boundary"
            ][
                "Python_process_startup_included_boolean"
            ]
            is False
        ),

    "jvm_startup_excluded":
        (
            protocol_remote[
                "measurement_boundary"
            ][
                "JVM_startup_included_boolean"
            ]
            is False
        ),

    "release_corpus_not_regenerated":
        (
            protocol_remote[
                "corpus_policy"
            ][
                "recreate_stage20_release_corpus_from_pcap"
            ]
            is False
        ),

    "complete_e2e_claim_blocked":
        (
            protocol_remote[
                "claim_boundary"
            ][
                "complete_end_to_end_claim_allowed"
            ]
            is False
        ),

    "gpu_off":
        (
            protocol_remote[
                "scientific_rules"
            ][
                "gpu_used"
            ]
            is False
        ),
}


for name, passed in checks.items():

    print(
        f"{name:40s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

    if not passed:

        raise RuntimeError(
            f"Remote scientific content audit failed: {name}"
        )


# =============================================================================
# 9. FINAL REPOSITORY AUDIT
# =============================================================================

banner(
    "STAGE26-4B1-GIT :: FINAL REPOSITORY AUDIT"
)


final_status = git(
    "status",
    "--porcelain",
)

final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_status:

    print(
        final_status
    )

    raise RuntimeError(
        "Repository not clean after Stage26-4B1 Git anchor."
    )


if final_head != final_remote:

    raise RuntimeError(
        "Local/remote divergence after Stage26-4B1 push."
    )


# =============================================================================
# 10. CLOSURE
# =============================================================================

banner(
    "STAGE26-4B1-GIT COMPLETE"
)


print(
    "Commit:"
)

print(
    " ",
    commit_sha
)

print(
    "Subject:"
)

print(
    " ",
    COMMIT_SUBJECT
)

print(
    "Parent:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nFrozen hashes:"
)

print(
    "  worker   :",
    EXPECTED_WORKER_SHA256
)

print(
    "  protocol :",
    EXPECTED_PROTOCOL_SHA256
)

print(
    "  boundary :",
    EXPECTED_BOUNDARY_SHA256
)

print(
    "  receipt  :",
    EXPECTED_FREEZE_RECEIPT_SHA256
)

print(
    "  manifest :",
    EXPECTED_MANIFEST_SHA256
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  local == origin/main       : PASS"
)

print(
    "  parent preserved           : PASS"
)

print(
    "  commit subject             : PASS"
)

print(
    "  manifest bytes             : PASS"
)

print(
    "  every manifest-listed file : PASS"
)

print(
    "  scientific content audit   : PASS"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  EXTRACTION PROTOCOL LOCKED REMOTELY : YES"
)

print(
    "  PCAP OPENED / ITERATED              : NO"
)

print(
    "  FULL-MONDAY VALIDATION              : NO"
)

print(
    "  EXTRACTION TIMING                   : NO"
)

print(
    "  CORPUS REGENERATED                  : NO"
)

print(
    "  MODELS LOADED                       : NO"
)

print(
    "  GPU                                 : NO"
)


print(
    "\nNEXT:"
)

print(
    "  STAGE26-4B2 — execute ONE full-Monday UNTIMED"
)

print(
    "  correctness validation against the frozen"
)

print(
    "  11,709,971-packet historical geometry."
)

print(
    "  Timed extraction remains FORBIDDEN until that gate passes."
)


STAGE26-4B1-GIT :: PRE-COMMIT GATE
Expected parent: 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Local HEAD     : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
origin/main    : 347d93f21d454cc5bda2c45889c890c67cdf0ecc

Git status:
?? results/stage26_deployment_profiling/stage26_4b_extraction_protocol_lock/

STAGE26-4B1-GIT :: LOCAL LOCK HASH GATE
stage26_raw_flow_extractor_v1.py                           PASS ba34c5ab1a8e0bdb22acf5d6be3430cc2b31fdec04dc333dcf3ef7344abe44e6
stage26_extraction_protocol.json                           PASS 12f7f8f0332d92694a827e3473dc1cef66939cf1cf1c49c17ad1699e60d929b0
stage26_component_boundary_map.json                        PASS 3f798bf9665dabe245e627b3a21ae0ecd2f837f84b091ad07bc13f60f97458cf
stage26_4b1_extraction_protocol_freeze_receipt.json        PASS 3152b3f35a8cf7920ce6d60f22de49f1f6fdc7bd36c1c35d763d10015f09f811
stage26_4b_extraction_lock_manifest.json                   PASS 7fe04d2628e1bfe7da5551192f45821b5406b3b3087976c4334bcd74224fc2d3

Manifest file

RuntimeError: $ git commit -m stage26: freeze extraction profiling protocol

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@21ba0ab9e27e.(none)')


In [14]:
# =============================================================================
# STAGE26-4B1-GIT RECOVERY
#
# FAILURE BEING RECOVERED:
#   git commit failed ONLY because fresh Kaggle session had no Git author
#   identity configured.
#
# THIS CELL:
#   - requires HEAD to still be the frozen parent;
#   - requires the exact Stage26-4B1 package to still be staged;
#   - derives author name/email from the existing durable parent commit;
#   - configures identity REPOSITORY-LOCALLY (not --global);
#   - commits the already-staged package;
#   - pushes using Kaggle GITHUB_TOKEN;
#   - fetches and remotely byte-verifies the frozen package.
#
# NO:
#   - PCAP access
#   - extraction
#   - timing
#   - corpus regeneration
#   - models
#   - holdout
#   - GPU
# =============================================================================

from __future__ import annotations

import os
import json
import hashlib
import stat
import subprocess
import tempfile
from pathlib import Path


# =============================================================================
# 0. FROZEN EXPECTATIONS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

LOCK_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_4b_extraction_protocol_lock"
)

LOCK_DIR = REPO / LOCK_REL


EXPECTED_PARENT = (
    "347d93f21d454cc5bda2c45889c890c67cdf0ecc"
)

COMMIT_SUBJECT = (
    "stage26: freeze extraction profiling protocol"
)


EXPECTED_HASHES = {
    "stage26_raw_flow_extractor_v1.py":
        "ba34c5ab1a8e0bdb22acf5d6be3430cc2b31fdec04dc333dcf3ef7344abe44e6",

    "stage26_extraction_protocol.json":
        "12f7f8f0332d92694a827e3473dc1cef66939cf1cf1c49c17ad1699e60d929b0",

    "stage26_component_boundary_map.json":
        "3f798bf9665dabe245e627b3a21ae0ecd2f837f84b091ad07bc13f60f97458cf",

    "stage26_4b1_extraction_protocol_freeze_receipt.json":
        "3152b3f35a8cf7920ce6d60f22de49f1f6fdc7bd36c1c35d763d10015f09f811",

    "stage26_4b_extraction_lock_manifest.json":
        "7fe04d2628e1bfe7da5551192f45821b5406b3b3087976c4334bcd74224fc2d3",
}


MANIFEST_REL = (
    LOCK_REL
    / "stage26_4b_extraction_lock_manifest.json"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 118
    )

    print(text)

    print(
        "=" * 118
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            + " ".join(
                map(
                    str,
                    cmd,
                )
            )
            + "\n\n"
            + output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        check=True,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def git_blob_bytes(
    treeish,
    relpath,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{relpath}",
        ],
        check=True,
        text=False,
    )

    return bytes(
        p.stdout
    )


# =============================================================================
# 2. INTERRUPTED-STATE GATE
# =============================================================================

banner(
    "STAGE26-4B1-GIT-RECOVERY :: INTERRUPTED STATE"
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

staged = git(
    "diff",
    "--cached",
    "--name-status",
)

unstaged = git(
    "diff",
    "--name-status",
)


print(
    "Expected HEAD:",
    EXPECTED_PARENT
)

print(
    "Local HEAD   :",
    head
)

print(
    "origin/main  :",
    remote
)


print(
    "\nStaged changes:"
)

print(
    staged
    if staged
    else "<none>"
)


print(
    "\nUnstaged tracked changes:"
)

print(
    unstaged
    if unstaged
    else "<none>"
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed; this recovery is only valid for the interrupted commit."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed since the interrupted commit."
    )


if not staged:

    raise RuntimeError(
        "The Stage26-4B1 lock package is no longer staged."
    )


if unstaged:

    raise RuntimeError(
        "Unexpected unstaged tracked modifications exist."
    )


expected_stage_paths = {
    str(
        LOCK_REL
        / "stage26_4b1_extraction_protocol_freeze_receipt.json"
    ),
    str(
        LOCK_REL
        / "stage26_4b_extraction_lock_manifest.json"
    ),
    str(
        LOCK_REL
        / "stage26_4b_upstream_provenance.json"
    ),
    str(
        LOCK_REL
        / "stage26_component_boundary_map.json"
    ),
    str(
        LOCK_REL
        / "stage26_extraction_protocol.json"
    ),
    str(
        LOCK_REL
        / "stage26_raw_flow_extractor_v1.py"
    ),
}


actual_stage_paths = set()


for line in staged.splitlines():

    parts = line.split(
        "\t",
        1,
    )

    if len(parts) != 2:

        raise RuntimeError(
            f"Unexpected staged line: {line}"
        )

    status_code, path = parts

    if status_code != "A":

        raise RuntimeError(
            f"Unexpected staged status {status_code}: {path}"
        )

    actual_stage_paths.add(
        path
    )


if actual_stage_paths != expected_stage_paths:

    raise RuntimeError(
        "Staged path set differs from the frozen Stage26-4B1 package."
    )


print(
    "\n[PASS] Exact six-file Stage26-4B1 package remains staged."
)


# =============================================================================
# 3. NARROW LOCAL HASH RECHECK
# =============================================================================

banner(
    "STAGE26-4B1-GIT-RECOVERY :: STAGED PACKAGE IDENTITY"
)


for filename, expected in EXPECTED_HASHES.items():

    path = LOCK_DIR / filename

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )

    print(
        f"{filename:58s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )

    if not passed:

        raise RuntimeError(
            f"Frozen lock file changed: {filename}"
        )


# =============================================================================
# 4. RESTORE REPOSITORY-LOCAL GIT IDENTITY
# =============================================================================

banner(
    "STAGE26-4B1-GIT-RECOVERY :: RESTORE GIT AUTHOR IDENTITY"
)


# Reuse the identity already present in the durable scientific history.
# No new identity/email is invented.

parent_author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_PARENT,
)

parent_author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_PARENT,
)


if not parent_author_name.strip():

    raise RuntimeError(
        "Could not recover parent commit author name."
    )


if (
    not parent_author_email.strip()
    or
    "@"
    not in parent_author_email
):

    raise RuntimeError(
        "Could not recover a valid parent commit author email."
    )


print(
    "Recovered author name :",
    parent_author_name
)

print(
    "Recovered author email:",
    parent_author_email
)


# IMPORTANT:
# repository-local config only; do not modify global Kaggle Git config.

git(
    "config",
    "--local",
    "user.name",
    parent_author_name,
)

git(
    "config",
    "--local",
    "user.email",
    parent_author_email,
)


configured_name = git(
    "config",
    "--local",
    "--get",
    "user.name",
)

configured_email = git(
    "config",
    "--local",
    "--get",
    "user.email",
)


print(
    "\nRepository-local user.name :",
    configured_name
)

print(
    "Repository-local user.email:",
    configured_email
)


if configured_name != parent_author_name:

    raise RuntimeError(
        "Repository-local Git name configuration failed."
    )


if configured_email != parent_author_email:

    raise RuntimeError(
        "Repository-local Git email configuration failed."
    )


print(
    "\n[PASS] Git identity restored locally for this repository only."
)


# =============================================================================
# 5. RESUME COMMIT
# =============================================================================

banner(
    "STAGE26-4B1-GIT-RECOVERY :: COMMIT"
)


commit_output = git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


print(
    commit_output
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Committed Stage26-4B1 package has wrong parent."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository is not clean after commit."
    )


# =============================================================================
# 6. PUSH USING KAGGLE SECRET
# =============================================================================

banner(
    "STAGE26-4B1-GIT-RECOVERY :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle secret GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    env = os.environ.copy()

    env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    p = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=env,
        check=True,
        text=True,
    )


    print(
        p.stdout.strip()
    )


github_token = None


# =============================================================================
# 7. FETCH / REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-4B1-GIT-RECOVERY :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed unexpectedly."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "Remote main does not equal new Stage26-4B1 commit."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote commit parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote commit subject mismatch."
    )


# =============================================================================
# 8. REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-4B1-GIT-RECOVERY :: REMOTE BYTE VERIFICATION"
)


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    str(
        MANIFEST_REL
    ),
)

remote_manifest_sha = hashlib.sha256(
    remote_manifest_bytes
).hexdigest()


print(
    "Manifest:"
)

print(
    " ",
    remote_manifest_sha
)


if (
    remote_manifest_sha
    !=
    EXPECTED_HASHES[
        "stage26_4b_extraction_lock_manifest.json"
    ]
):

    raise RuntimeError(
        "Remote manifest SHA mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


for row in remote_manifest[
    "files"
]:

    rel = row[
        "repo_relative_path"
    ]

    data = git_blob_bytes(
        "origin/main",
        rel,
    )

    actual_size = len(
        data
    )

    actual_sha = hashlib.sha256(
        data
    ).hexdigest()


    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{rel}"
    )


    if not passed:

        raise RuntimeError(
            f"Remote byte verification failed: {rel}"
        )


print(
    "\n[PASS] Every manifest-listed file is byte-identical on origin/main."
)


# =============================================================================
# 9. FINAL AUDIT
# =============================================================================

banner(
    "STAGE26-4B1-GIT-RECOVERY :: FINAL AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != final_remote:

    raise RuntimeError(
        "Local/remote divergence."
    )


if final_status:

    raise RuntimeError(
        "Repository not clean after recovery."
    )


banner(
    "STAGE26-4B1-GIT RECOVERY COMPLETE"
)


print(
    "COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nSUBJECT:"
)

print(
    " ",
    COMMIT_SUBJECT
)


print(
    "\nPARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nREMOTE ANCHOR:"
)

print(
    "  local == origin/main       : PASS"
)

print(
    "  parent                     : PASS"
)

print(
    "  frozen manifest            : PASS"
)

print(
    "  manifest-listed bytes      : PASS"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  EXTRACTION PROTOCOL REMOTELY LOCKED : YES"
)

print(
    "  PCAP OPENED / ITERATED             : NO"
)

print(
    "  FULL-MONDAY VALIDATION             : NO"
)

print(
    "  EXTRACTION TIMING                  : NO"
)

print(
    "  RELEASE CORPUS REGENERATED         : NO"
)

print(
    "  GPU                                : NO"
)


print(
    "\nNEXT:"
)

print(
    "  STAGE26-4B2 — ONE full-Monday UNTIMED correctness"
)

print(
    "  validation against the frozen Stage20 raw geometry."
)

print(
    "  Timed extraction remains forbidden until it passes."
)


STAGE26-4B1-GIT-RECOVERY :: INTERRUPTED STATE
Expected HEAD: 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Local HEAD   : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
origin/main  : 347d93f21d454cc5bda2c45889c890c67cdf0ecc

Staged changes:
A	results/stage26_deployment_profiling/stage26_4b_extraction_protocol_lock/stage26_4b1_extraction_protocol_freeze_receipt.json
A	results/stage26_deployment_profiling/stage26_4b_extraction_protocol_lock/stage26_4b_extraction_lock_manifest.json
A	results/stage26_deployment_profiling/stage26_4b_extraction_protocol_lock/stage26_4b_upstream_provenance.json
A	results/stage26_deployment_profiling/stage26_4b_extraction_protocol_lock/stage26_component_boundary_map.json
A	results/stage26_deployment_profiling/stage26_4b_extraction_protocol_lock/stage26_extraction_protocol.json
A	results/stage26_deployment_profiling/stage26_4b_extraction_protocol_lock/stage26_raw_flow_extractor_v1.py

Unstaged tracked changes:
<none>

[PASS] Exact six-file Stage26-4B1 package remains st

In [15]:
# =============================================================================
# STAGE26-4B2 — FULL MONDAY UNTIMED CORRECTNESS VALIDATION
#
# DURABLE PROTOCOL COMMIT:
#   b41a0559598a70e6c69aac05c76e4c4edb73b34e
#
# THIS IS THE FIRST STAGE26-4 CELL ALLOWED TO ITERATE THE PCAP.
#
# PURPOSE:
#   Execute the remotely-frozen source-faithful extractor ONCE over the
#   complete exact Monday PCAP and require exact reproduction of every
#   historical Stage20 raw reconstruction anchor.
#
# ABSOLUTE RULES:
#   - FULL Monday capture only.
#   - NO performance timer.
#   - NO extraction throughput measurement.
#   - NO corpus creation/regeneration.
#   - NO CSV generation.
#   - NO labels.
#   - NO models.
#   - NO Thursday/Friday.
#   - NO GPU.
#   - NO Git write.
#
# IMPORTANT:
#   The worker intentionally prints no progress during the full scan.
#   A quiet cell for several minutes is expected.
#
# IF ANY COUNT DIFFERS:
#   - timing remains FORBIDDEN;
#   - write a local mismatch receipt;
#   - stop for narrow parser/lifecycle diagnosis;
#   - DO NOT change the frozen source/sample/repetitions/timing protocol.
# =============================================================================

from __future__ import annotations

import os
import sys
import json
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. FROZEN IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

EXTRACTION_ROOT = (
    STAGE26_ROOT
    / "extraction"
)

VALIDATION_ROOT = (
    EXTRACTION_ROOT
    / "stage26_4b2_full_monday_validation"
)

VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


EXPECTED_HEAD = (
    "b41a0559598a70e6c69aac05c76e4c4edb73b34e"
)

EXPECTED_PARENT = (
    "347d93f21d454cc5bda2c45889c890c67cdf0ecc"
)


LOCK_ROOT = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4b_extraction_protocol_lock"
)


WORKER = (
    LOCK_ROOT
    / "stage26_raw_flow_extractor_v1.py"
)

PROTOCOL = (
    LOCK_ROOT
    / "stage26_extraction_protocol.json"
)

MANIFEST = (
    LOCK_ROOT
    / "stage26_4b_extraction_lock_manifest.json"
)

FREEZE_RECEIPT = (
    LOCK_ROOT
    / "stage26_4b1_extraction_protocol_freeze_receipt.json"
)


EXPECTED_WORKER_SHA256 = (
    "ba34c5ab1a8e0bdb22acf5d6be3430cc2b31fdec04dc333dcf3ef7344abe44e6"
)

EXPECTED_PROTOCOL_SHA256 = (
    "12f7f8f0332d92694a827e3473dc1cef66939cf1cf1c49c17ad1699e60d929b0"
)

EXPECTED_MANIFEST_SHA256 = (
    "7fe04d2628e1bfe7da5551192f45821b5406b3b3087976c4334bcd74224fc2d3"
)

EXPECTED_FREEZE_RECEIPT_SHA256 = (
    "3152b3f35a8cf7920ce6d60f22de49f1f6fdc7bd36c1c35d763d10015f09f811"
)


# -----------------------------------------------------------------------------
# Exact Monday source
# -----------------------------------------------------------------------------

MONDAY_PCAP = (
    STAGE26_ROOT
    / "sources"
    / "Monday-WorkingHours.pcap"
)

EXPECTED_PCAP_SIZE = (
    10_822_507_416
)

EXPECTED_PCAP_SHA256 = (
    "f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972"
)


SOURCE_RECEIPT = (
    EXTRACTION_ROOT
    / "stage26_4a_monday_source_restoration_receipt.json"
)

EXPECTED_SOURCE_RECEIPT_SHA256 = (
    "a7d25f504c5a1d75f6cb46b94bd498ae6ca236a3872be5f868ce9160ec269bb3"
)


# -----------------------------------------------------------------------------
# FULL historical Stage20 exact anchors
# -----------------------------------------------------------------------------

EXPECTED_COUNTS = {
    "raw_packet_count":
        11_709_971,

    "valid_ipv4_packet_count":
        11_626_492,

    "non_ipv4_packet_count":
        83_479,

    "parser_tcp_count":
        10_718_469,

    "parser_udp_count":
        907_039,

    "parser_other0_count":
        984,

    "exportable_flow_count":
        529_601,

    "retained_packet_count":
        11_573_331,

    "max_active_flows":
        246_086,

    "fin_exportable_flows":
        216_388,

    "timeout_exportable_flows":
        120_017,

    "eof_current_exportable_flows":
        193_196,

    "discarded_singleton_timeout_flows":
        271,

    "eof_singleton_discarded_flows":
        52_890,
}


EXPECTED_PARSER_ID = (
    "STAGE20_FROZEN_IPV4_DIRECT_TRANSPORT_TCP6_UDP17_OTHER0"
)

EXPECTED_FLOW_ID = (
    "BASICPACKETINFO_JAVA_SIGNED_IPV4_FIRST_DIFFERING_BYTE_"
    "SWAP_IPS_AND_PORTS_TOGETHER"
)

EXPECTED_LIFECYCLE_ID = (
    "CICFLOWMETER_FLOWGENERATOR_"
    "EAA853DD82F08BA5288BB7F295B471DE7313F883"
)


# =============================================================================
# 1. OUTPUTS
# =============================================================================

CONFIG_PATH = (
    VALIDATION_ROOT
    / "stage26_4b2_full_monday_validation_config.json"
)

RESULT_PATH = (
    VALIDATION_ROOT
    / "stage26_4b2_full_monday_validation_result.json"
)

RECEIPT_PATH = (
    VALIDATION_ROOT
    / "stage26_4b2_full_monday_validation_receipt.json"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 122
    )

    print(text)

    print(
        "=" * 122
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            + " ".join(
                map(
                    str,
                    cmd,
                )
            )
            + "\n\n"
            + p.stdout
        )

    return p


def git(*args):

    return run(
        [
            "git",
            *args,
        ]
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


# =============================================================================
# 3. DURABLE GIT / PROTOCOL GATE
# =============================================================================

banner(
    "STAGE26-4B2 :: DURABLE PROTOCOL GATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

parent = git(
    "rev-parse",
    "HEAD^",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD :",
    EXPECTED_HEAD
)

print(
    "Local HEAD    :",
    head
)

print(
    "origin/main   :",
    remote
)

print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Actual parent  :",
    parent
)

print(
    "Repo clean    :",
    status == ""
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected Stage26-4B1 durable commit."
    )


if remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main no longer equals Stage26-4B1."
    )


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-4B1 parent changed."
    )


if status:

    raise RuntimeError(
        "Repository must be clean before full validation."
    )


# =============================================================================
# 4. FROZEN FILE HASH GATE
# =============================================================================

banner(
    "STAGE26-4B2 :: FROZEN FILE HASH GATE"
)


hash_checks = [
    (
        "worker",
        WORKER,
        EXPECTED_WORKER_SHA256,
    ),
    (
        "protocol",
        PROTOCOL,
        EXPECTED_PROTOCOL_SHA256,
    ),
    (
        "manifest",
        MANIFEST,
        EXPECTED_MANIFEST_SHA256,
    ),
    (
        "freeze receipt",
        FREEZE_RECEIPT,
        EXPECTED_FREEZE_RECEIPT_SHA256,
    ),
    (
        "source restoration receipt",
        SOURCE_RECEIPT,
        EXPECTED_SOURCE_RECEIPT_SHA256,
    ),
]


for name, path, expected in hash_checks:

    if not path.exists():

        raise FileNotFoundError(
            path
        )


    actual = sha256_file(
        path
    )


    passed = (
        actual == expected
    )


    print(
        f"{name:28s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Frozen hash mismatch: {name}"
        )


# =============================================================================
# 5. FROZEN PROTOCOL CONTENT GATE
# =============================================================================

banner(
    "STAGE26-4B2 :: PROTOCOL CONTENT GATE"
)


protocol = json.loads(
    PROTOCOL.read_text(
        encoding="utf-8"
    )
)


if (
    protocol[
        "status"
    ]
    !=
    "FROZEN_BEFORE_FIRST_EXTRACTION_MEASUREMENT"
):

    raise RuntimeError(
        "Unexpected extraction protocol state."
    )


correctness_gate = protocol[
    "full_untimed_correctness_gate"
]


if (
    correctness_gate[
        "required_before_any_timed_extraction"
    ]
    is not True
):

    raise RuntimeError(
        "Full validation is not frozen as mandatory."
    )


if (
    correctness_gate[
        "timing_performed"
    ]
    is not False
):

    raise RuntimeError(
        "Frozen correctness gate unexpectedly permits timing."
    )


protocol_expected_counts = correctness_gate[
    "expected_exact_counts"
]


if protocol_expected_counts != EXPECTED_COUNTS:

    print(
        "Protocol expected counts:"
    )

    print(
        json.dumps(
            protocol_expected_counts,
            indent=2,
            sort_keys=True,
        )
    )

    raise RuntimeError(
        "Notebook expected-count constants differ from frozen protocol."
    )


print(
    "Full validation required : PASS"
)

print(
    "Performance timing       : FORBIDDEN"
)

print(
    "Expected anchors loaded  :",
    len(
        EXPECTED_COUNTS
    )
)


# =============================================================================
# 6. SOURCE IDENTITY GATE — DO NOT REHASH 10 GiB
# =============================================================================

banner(
    "STAGE26-4B2 :: SOURCE IDENTITY GATE"
)


if not MONDAY_PCAP.exists():

    raise FileNotFoundError(
        MONDAY_PCAP
    )


resolved_source = MONDAY_PCAP.resolve(
    strict=True
)


actual_source_size = int(
    resolved_source.stat().st_size
)


source_receipt = json.loads(
    SOURCE_RECEIPT.read_text(
        encoding="utf-8"
    )
)


verified_sha = source_receipt[
    "recovery"
][
    "verified_sha256"
]


print(
    "Canonical source:"
)

print(
    " ",
    MONDAY_PCAP
)

print(
    "Resolved source:"
)

print(
    " ",
    resolved_source
)

print(
    "Size bytes:"
)

print(
    " ",
    actual_source_size
)

print(
    "Previously full-verified SHA256:"
)

print(
    " ",
    verified_sha
)


if actual_source_size != EXPECTED_PCAP_SIZE:

    raise RuntimeError(
        "Monday PCAP size changed."
    )


if verified_sha != EXPECTED_PCAP_SHA256:

    raise RuntimeError(
        "Monday PCAP identity receipt mismatch."
    )


print(
    "\n[PASS] Existing exact Monday source identity preserved."
)

print(
    "Full 10.08-GiB SHA is NOT redundantly recomputed."
)


# =============================================================================
# 7. FIRST-RUN / NON-OVERWRITE GATE
# =============================================================================

banner(
    "STAGE26-4B2 :: NON-OVERWRITE GATE"
)


existing_outputs = [
    p
    for p in [
        CONFIG_PATH,
        RESULT_PATH,
        RECEIPT_PATH,
    ]
    if p.exists()
]


if existing_outputs:

    print(
        "Existing Stage26-4B2 outputs:"
    )

    for p in existing_outputs:

        print(
            " ",
            p
        )

    raise RuntimeError(
        "Stage26-4B2 validation outputs already exist. "
        "Do not overwrite or remeasure."
    )


print(
    "No prior Stage26-4B2 validation output exists: PASS"
)


# =============================================================================
# 8. WRITE EXACT UNTIMED WORKER CONFIG
# =============================================================================

banner(
    "STAGE26-4B2 :: WRITE UNTIMED VALIDATION CONFIG"
)


config = {
    "schema":
        "stage26_4b2_full_monday_validation_config_v1",

    "mode":
        "FULL_VALIDATION_UNTIMED",

    "pcap_path":
        str(
            MONDAY_PCAP
        ),

    "result_path":
        str(
            RESULT_PATH
        ),

    "affinity":
        [
            0
        ],

    "packet_limit":
        None,

    "performance_timing_allowed":
        False,

    "corpus_creation_allowed":
        False,

    "gpu_allowed":
        False,

    "protocol_commit":
        EXPECTED_HEAD,

    "protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "worker_sha256":
        EXPECTED_WORKER_SHA256,

    "source_sha256":
        EXPECTED_PCAP_SHA256,
}


atomic_json(
    CONFIG_PATH,
    config,
)


config_sha = sha256_file(
    CONFIG_PATH
)


print(
    "Config:"
)

print(
    " ",
    CONFIG_PATH
)

print(
    "SHA256:"
)

print(
    " ",
    config_sha
)


# =============================================================================
# 9. EXECUTE ONE FULL-MONDAY UNTIMED VALIDATION
# =============================================================================

banner(
    "STAGE26-4B2 :: FULL MONDAY VALIDATION START"
)


print(
    "Worker:"
)

print(
    " ",
    WORKER
)

print(
    "\nScope:"
)

print(
    "  ALL 11,709,971 raw Monday packet records"
)

print(
    "\nPerformance timer:"
)

print(
    "  NONE"
)

print(
    "\nExpected behavior:"
)

print(
    "  This cell may remain silent for several minutes."
)

print(
    "  Do not interrupt unless Kaggle itself reports a failure."
)

print(
    "\nStarting frozen worker now...",
    flush=True,
)


worker_env = os.environ.copy()


# Explicit CPU-only execution context.
worker_env[
    "CUDA_VISIBLE_DEVICES"
] = ""


worker_env[
    "PYTHONHASHSEED"
] = "0"


# Prevent numerical/runtime libraries from opportunistically spawning
# additional threads. The extractor itself is Python/struct/dict based,
# but this makes the execution envelope explicit.
for env_name in [
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
]:

    worker_env[
        env_name
    ] = "1"


# CRITICAL:
#   No parent performance timer.
#   No timeout is imposed here because this is a mandatory correctness pass,
#   not one of the frozen timed extraction repetitions.
worker_process = subprocess.run(
    [
        sys.executable,
        str(
            WORKER
        ),
        str(
            CONFIG_PATH
        ),
    ],
    cwd=REPO,
    env=worker_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)


print(
    "\nFrozen worker returned."
)

print(
    "Return code:",
    worker_process.returncode
)


if worker_process.stdout.strip():

    print(
        "\nWorker output:"
    )

    print(
        worker_process.stdout
    )


if worker_process.returncode != 0:

    raise RuntimeError(
        "Full-Monday untimed worker failed before producing a valid result."
    )


if not RESULT_PATH.exists():

    raise RuntimeError(
        "Worker returned successfully but validation result is missing."
    )


# =============================================================================
# 10. RESULT IDENTITY / MODE GATE
# =============================================================================

banner(
    "STAGE26-4B2 :: RESULT MODE GATE"
)


result_sha = sha256_file(
    RESULT_PATH
)


result = json.loads(
    RESULT_PATH.read_text(
        encoding="utf-8"
    )
)


print(
    "Result:"
)

print(
    " ",
    RESULT_PATH
)

print(
    "SHA256:"
)

print(
    " ",
    result_sha
)


mode_checks = {
    "status_PASS":
        (
            result.get(
                "status"
            )
            ==
            "PASS"
        ),

    "mode_FULL_VALIDATION_UNTIMED":
        (
            result.get(
                "mode"
            )
            ==
            "FULL_VALIDATION_UNTIMED"
        ),

    "timing_performed_FALSE":
        (
            result.get(
                "timing_performed"
            )
            is False
        ),

    "packet_limit_NONE":
        (
            result.get(
                "packet_limit"
            )
            is None
        ),

    "disk_IO_included_TRUE":
        (
            result.get(
                "disk_IO_included"
            )
            is True
        ),

    "serialization_included_FALSE":
        (
            result.get(
                "serialization_included"
            )
            is False
        ),

    "Python_startup_included_FALSE":
        (
            result.get(
                "Python_process_startup_included"
            )
            is False
        ),

    "JVM_startup_included_FALSE":
        (
            result.get(
                "JVM_startup_included"
            )
            is False
        ),

    "corpus_created_or_modified_FALSE":
        (
            result.get(
                "corpus_created_or_modified"
            )
            is False
        ),

    "labels_accessed_FALSE":
        (
            result.get(
                "labels_accessed"
            )
            is False
        ),

    "gpu_used_FALSE":
        (
            result.get(
                "gpu_used"
            )
            is False
        ),

    "parser_semantics":
        (
            result.get(
                "parser_semantics_identifier"
            )
            ==
            EXPECTED_PARSER_ID
        ),

    "flow_id_semantics":
        (
            result.get(
                "flow_id_semantics_identifier"
            )
            ==
            EXPECTED_FLOW_ID
        ),

    "flow_lifecycle_semantics":
        (
            result.get(
                "flow_lifecycle_semantics_identifier"
            )
            ==
            EXPECTED_LIFECYCLE_ID
        ),
}


for name, passed in mode_checks.items():

    print(
        f"{name:42s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    mode_checks.values()
):

    raise RuntimeError(
        "Full-validation result mode/semantics gate failed."
    )


# =============================================================================
# 11. COUNT FINGERPRINT INTEGRITY
# =============================================================================

banner(
    "STAGE26-4B2 :: COUNT FINGERPRINT INTEGRITY"
)


actual_counts = result[
    "counts"
]


recomputed_count_sha = hashlib.sha256(
    json.dumps(
        actual_counts,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


recorded_count_sha = result[
    "count_summary_sha256"
]


print(
    "Recorded count SHA256:"
)

print(
    " ",
    recorded_count_sha
)

print(
    "Recomputed count SHA256:"
)

print(
    " ",
    recomputed_count_sha
)


if recorded_count_sha != recomputed_count_sha:

    raise RuntimeError(
        "Validation count fingerprint mismatch."
    )


print(
    "\n[PASS] Result count dictionary integrity."
)


# =============================================================================
# 12. EXACT HISTORICAL COUNT COMPARISON
# =============================================================================

banner(
    "STAGE26-4B2 :: EXACT STAGE20 HISTORICAL COUNT GATE"
)


comparison_rows = []

mismatches = []


for key, expected in EXPECTED_COUNTS.items():

    if key not in actual_counts:

        actual = None

        passed = False

    else:

        actual = actual_counts[
            key
        ]

        passed = (
            int(
                actual
            )
            ==
            int(
                expected
            )
        )


    comparison_rows.append(
        {
            "metric":
                key,

            "expected":
                expected,

            "actual":
                actual,

            "pass":
                passed,
        }
    )


    print(
        f"{key:42s} "
        f"expected={expected:>12,d}  "
        f"actual="
        +
        (
            f"{int(actual):>12,d}"
            if actual is not None
            else f"{'MISSING':>12s}"
        )
        +
        f"  {'PASS' if passed else 'FAIL'}"
    )


    if not passed:

        mismatches.append(
            {
                "metric":
                    key,

                "expected":
                    expected,

                "actual":
                    actual,
            }
        )


# =============================================================================
# 13. INTERNAL ARITHMETIC CONSISTENCY
# =============================================================================

banner(
    "STAGE26-4B2 :: INTERNAL ARITHMETIC AUDIT"
)


arithmetic_checks = {
    "raw = IPv4 + nonIPv4":
        (
            int(
                actual_counts[
                    "raw_packet_count"
                ]
            )
            ==
            int(
                actual_counts[
                    "valid_ipv4_packet_count"
                ]
            )
            +
            int(
                actual_counts[
                    "non_ipv4_packet_count"
                ]
            )
        ),

    "IPv4 = TCP + UDP + OTHER0":
        (
            int(
                actual_counts[
                    "valid_ipv4_packet_count"
                ]
            )
            ==
            int(
                actual_counts[
                    "parser_tcp_count"
                ]
            )
            +
            int(
                actual_counts[
                    "parser_udp_count"
                ]
            )
            +
            int(
                actual_counts[
                    "parser_other0_count"
                ]
            )
        ),

    "flows = FIN + timeout + EOF":
        (
            int(
                actual_counts[
                    "exportable_flow_count"
                ]
            )
            ==
            int(
                actual_counts[
                    "fin_exportable_flows"
                ]
            )
            +
            int(
                actual_counts[
                    "timeout_exportable_flows"
                ]
            )
            +
            int(
                actual_counts[
                    "eof_current_exportable_flows"
                ]
            )
        ),
}


for name, passed in arithmetic_checks.items():

    print(
        f"{name:42s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    arithmetic_checks.values()
):

    raise RuntimeError(
        "Internal full-validation arithmetic consistency failed."
    )


# =============================================================================
# 14. WRITE LOCAL VALIDATION RECEIPT
# =============================================================================

validation_passed = (
    len(
        mismatches
    )
    ==
    0
)


validation_status = (
    "PASS_EXACT_STAGE20_RECONSTRUCTION"
    if validation_passed
    else
    "FAIL_STAGE20_RECONSTRUCTION_MISMATCH"
)


receipt = {
    "schema":
        "stage26_4b2_full_monday_validation_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4B2",

    "status":
        validation_status,

    "protocol_commit":
        EXPECTED_HEAD,

    "protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "worker_sha256":
        EXPECTED_WORKER_SHA256,

    "source_sha256":
        EXPECTED_PCAP_SHA256,

    "source_size_bytes":
        EXPECTED_PCAP_SIZE,

    "config_path":
        str(
            CONFIG_PATH
        ),

    "config_sha256":
        config_sha,

    "result_path":
        str(
            RESULT_PATH
        ),

    "result_sha256":
        result_sha,

    "count_summary_sha256":
        recorded_count_sha,

    "exact_comparison":
        comparison_rows,

    "mismatches":
        mismatches,

    "arithmetic_checks":
        arithmetic_checks,

    "all_frozen_counts_exact":
        validation_passed,

    "timing_performed":
        False,

    "performance_metrics_generated":
        False,

    "corpus_created_or_modified":
        False,

    "release_corpus_regenerated":
        False,

    "labels_accessed":
        False,

    "models_loaded":
        False,

    "Thursday_accessed":
        False,

    "Friday_accessed":
        False,

    "gpu_used":
        False,

    "timed_extraction_allowed_next":
        validation_passed,

    "failure_rule":
        (
            None
            if validation_passed
            else
            (
                "Timed extraction remains forbidden. "
                "Perform narrow source-faithful parser/lifecycle diagnosis only."
            )
        ),
}


atomic_json(
    RECEIPT_PATH,
    receipt,
)


receipt_sha = sha256_file(
    RECEIPT_PATH
)


# =============================================================================
# 15. FINAL REPOSITORY AUDIT
# =============================================================================

banner(
    "STAGE26-4B2 :: FINAL REPOSITORY AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_HEAD:

    raise RuntimeError(
        "Git HEAD changed during validation."
    )


if final_remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed during validation."
    )


if final_status:

    print(
        final_status
    )

    raise RuntimeError(
        "Repository changed during full validation."
    )


# =============================================================================
# 16. CLOSURE
# =============================================================================

banner(
    "STAGE26-4B2 FULL MONDAY UNTIMED VALIDATION COMPLETE"
)


print(
    "VALIDATION STATUS:"
)

print(
    " ",
    validation_status
)


print(
    "\nRESULT SHA256:"
)

print(
    " ",
    result_sha
)


print(
    "\nCOUNT FINGERPRINT:"
)

print(
    " ",
    recorded_count_sha
)


print(
    "\nVALIDATION RECEIPT:"
)

print(
    " ",
    RECEIPT_PATH
)

print(
    " SHA256:",
    receipt_sha
)


print(
    "\nEXACT HISTORICAL ANCHORS:"
)

print(
    "  checked    :",
    len(
        EXPECTED_COUNTS
    )
)

print(
    "  mismatches :",
    len(
        mismatches
    )
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  FULL MONDAY PCAP ITERATED          : YES"
)

print(
    "  PERFORMANCE TIMING                 : NO"
)

print(
    "  EXTRACTION THROUGHPUT MEASURED     : NO"
)

print(
    "  CORPUS CREATED                     : NO"
)

print(
    "  RELEASE CORPUS REGENERATED         : NO"
)

print(
    "  LABELS ACCESSED                    : NO"
)

print(
    "  MODELS LOADED                      : NO"
)

print(
    "  GPU                                : NO"
)


if validation_passed:

    print(
        "\n[PASS] Frozen Stage26 extractor exactly reproduces"
    )

    print(
        "       every declared Stage20 Monday raw reconstruction anchor."
    )

    print(
        "\nNEXT:"
    )

    print(
        "  Anchor this untimed correctness result to GitHub BEFORE"
    )

    print(
        "  executing the five frozen 1,000,000-packet timed repetitions."
    )


else:

    print(
        "\n[FAIL] FULL-MONDAY CORRECTNESS GATE DID NOT PASS."
    )

    print(
        "Timed extraction remains FORBIDDEN."
    )

    print(
        "Next action is narrow parser/lifecycle diagnosis only."
    )

    raise RuntimeError(
        "Stage26-4B2 exact historical correctness gate failed."
    )


STAGE26-4B2 :: DURABLE PROTOCOL GATE
Expected HEAD : b41a0559598a70e6c69aac05c76e4c4edb73b34e
Local HEAD    : b41a0559598a70e6c69aac05c76e4c4edb73b34e
origin/main   : b41a0559598a70e6c69aac05c76e4c4edb73b34e
Expected parent: 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Actual parent  : 347d93f21d454cc5bda2c45889c890c67cdf0ecc
Repo clean    : True

STAGE26-4B2 :: FROZEN FILE HASH GATE
worker                       PASS ba34c5ab1a8e0bdb22acf5d6be3430cc2b31fdec04dc333dcf3ef7344abe44e6
protocol                     PASS 12f7f8f0332d92694a827e3473dc1cef66939cf1cf1c49c17ad1699e60d929b0
manifest                     PASS 7fe04d2628e1bfe7da5551192f45821b5406b3b3087976c4334bcd74224fc2d3
freeze receipt               PASS 3152b3f35a8cf7920ce6d60f22de49f1f6fdc7bd36c1c35d763d10015f09f811
source restoration receipt   PASS a7d25f504c5a1d75f6cb46b94bd498ae6ca236a3872be5f868ce9160ec269bb3

STAGE26-4B2 :: PROTOCOL CONTENT GATE
Full validation required : PASS
Performance timing       : FORBIDDEN
Expected ancho

In [16]:
# =============================================================================
# STAGE26-4B2-GIT
# ANCHOR FULL-MONDAY UNTIMED CORRECTNESS VALIDATION
#
# CURRENT DURABLE PARENT:
#   b41a0559598a70e6c69aac05c76e4c4edb73b34e
#
# VALIDATION ALREADY COMPLETED:
#   14 / 14 frozen Stage20 historical anchors EXACT
#   0 mismatches
#
# THIS CELL ONLY:
#   - verifies the existing validation outputs;
#   - copies them into a durable repository checkpoint;
#   - writes a package manifest;
#   - commits;
#   - pushes;
#   - remotely byte-verifies the committed checkpoint.
#
# ABSOLUTE RULES:
#   - DO NOT rerun extraction.
#   - DO NOT open or hash the 10.08-GiB PCAP.
#   - DO NOT perform timing.
#   - DO NOT regenerate any corpus.
#   - DO NOT load models.
#   - DO NOT access holdout / Thursday / Friday.
#   - GPU remains OFF.
# =============================================================================

from __future__ import annotations

import os
import json
import hashlib
import shutil
import stat
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. FROZEN EXPECTATIONS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

SOURCE_VALIDATION_DIR = (
    STAGE26_ROOT
    / "extraction"
    / "stage26_4b2_full_monday_validation"
)

SOURCE_CONFIG = (
    SOURCE_VALIDATION_DIR
    / "stage26_4b2_full_monday_validation_config.json"
)

SOURCE_RESULT = (
    SOURCE_VALIDATION_DIR
    / "stage26_4b2_full_monday_validation_result.json"
)

SOURCE_RECEIPT = (
    SOURCE_VALIDATION_DIR
    / "stage26_4b2_full_monday_validation_receipt.json"
)


EXPECTED_PARENT = (
    "b41a0559598a70e6c69aac05c76e4c4edb73b34e"
)

EXPECTED_PROTOCOL_PARENT = (
    "347d93f21d454cc5bda2c45889c890c67cdf0ecc"
)


EXPECTED_CONFIG_SHA256 = (
    "409f43dfe167b17595b101b37b92d0796c757fd7cdb8c0946d9969f3c61ce122"
)

EXPECTED_RESULT_SHA256 = (
    "a012ba81b4cbb339371898e6adc4cd43a1c6364086d1b343f0fb7cd73c3bb46f"
)

EXPECTED_RECEIPT_SHA256 = (
    "0634cd71d3b1d2111a7b44a3519711660fb6ec3ad8f143804ba1cd3b9e6802d2"
)

EXPECTED_COUNT_FINGERPRINT = (
    "2ac0f791b97187cb10a4228775146958830719b29dc10a064ed5adae0614728e"
)


EXPECTED_PROTOCOL_SHA256 = (
    "12f7f8f0332d92694a827e3473dc1cef66939cf1cf1c49c17ad1699e60d929b0"
)

EXPECTED_WORKER_SHA256 = (
    "ba34c5ab1a8e0bdb22acf5d6be3430cc2b31fdec04dc333dcf3ef7344abe44e6"
)


EXPECTED_PCAP_SHA256 = (
    "f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972"
)

EXPECTED_PCAP_SIZE = (
    10_822_507_416
)


EXPECTED_COUNTS = {
    "raw_packet_count":
        11_709_971,

    "valid_ipv4_packet_count":
        11_626_492,

    "non_ipv4_packet_count":
        83_479,

    "parser_tcp_count":
        10_718_469,

    "parser_udp_count":
        907_039,

    "parser_other0_count":
        984,

    "exportable_flow_count":
        529_601,

    "retained_packet_count":
        11_573_331,

    "max_active_flows":
        246_086,

    "fin_exportable_flows":
        216_388,

    "timeout_exportable_flows":
        120_017,

    "eof_current_exportable_flows":
        193_196,

    "discarded_singleton_timeout_flows":
        271,

    "eof_singleton_discarded_flows":
        52_890,
}


COMMIT_SUBJECT = (
    "stage26: anchor full Monday extraction validation"
)


# =============================================================================
# 1. DURABLE OUTPUT PACKAGE
# =============================================================================

CHECKPOINT_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_4b2_full_monday_validation"
)

CHECKPOINT_DIR = (
    REPO
    / CHECKPOINT_REL
)

MANIFEST = (
    CHECKPOINT_DIR
    / "stage26_4b2_validation_manifest.json"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 122
    )

    print(text)

    print(
        "=" * 122
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        check=True,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        check=True,
        text=False,
    )

    return bytes(
        p.stdout
    )


# =============================================================================
# 3. PRE-ANCHOR GIT GATE
# =============================================================================

banner(
    "STAGE26-4B2-GIT :: PRE-ANCHOR GIT GATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected local HEAD before Stage26-4B2 anchor."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Stage26-4B2 anchor."
    )


if status:

    raise RuntimeError(
        "Repository must be clean before Stage26-4B2 anchor."
    )


# =============================================================================
# 4. VALIDATION OUTPUT IDENTITY GATE
# =============================================================================

banner(
    "STAGE26-4B2-GIT :: VALIDATION OUTPUT IDENTITY"
)


expected_files = [
    (
        "config",
        SOURCE_CONFIG,
        EXPECTED_CONFIG_SHA256,
    ),
    (
        "result",
        SOURCE_RESULT,
        EXPECTED_RESULT_SHA256,
    ),
    (
        "receipt",
        SOURCE_RECEIPT,
        EXPECTED_RECEIPT_SHA256,
    ),
]


for name, path, expected in expected_files:

    if not path.exists():

        raise FileNotFoundError(
            path
        )

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )

    print(
        f"{name:12s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )

    if not passed:

        raise RuntimeError(
            f"Stage26-4B2 {name} changed."
        )


# =============================================================================
# 5. SCIENTIFIC CONTENT GATE
# =============================================================================

banner(
    "STAGE26-4B2-GIT :: SCIENTIFIC CONTENT GATE"
)


config = json.loads(
    SOURCE_CONFIG.read_text(
        encoding="utf-8"
    )
)

result = json.loads(
    SOURCE_RESULT.read_text(
        encoding="utf-8"
    )
)

receipt = json.loads(
    SOURCE_RECEIPT.read_text(
        encoding="utf-8"
    )
)


checks = {
    "config_mode_untimed":
        (
            config[
                "mode"
            ]
            ==
            "FULL_VALIDATION_UNTIMED"
        ),

    "config_protocol_commit":
        (
            config[
                "protocol_commit"
            ]
            ==
            EXPECTED_PARENT
        ),

    "config_protocol_sha":
        (
            config[
                "protocol_sha256"
            ]
            ==
            EXPECTED_PROTOCOL_SHA256
        ),

    "config_worker_sha":
        (
            config[
                "worker_sha256"
            ]
            ==
            EXPECTED_WORKER_SHA256
        ),

    "config_source_sha":
        (
            config[
                "source_sha256"
            ]
            ==
            EXPECTED_PCAP_SHA256
        ),

    "result_status_pass":
        (
            result[
                "status"
            ]
            ==
            "PASS"
        ),

    "result_mode_untimed":
        (
            result[
                "mode"
            ]
            ==
            "FULL_VALIDATION_UNTIMED"
        ),

    "result_timing_false":
        (
            result[
                "timing_performed"
            ]
            is False
        ),

    "result_count_fingerprint":
        (
            result[
                "count_summary_sha256"
            ]
            ==
            EXPECTED_COUNT_FINGERPRINT
        ),

    "result_corpus_not_modified":
        (
            result[
                "corpus_created_or_modified"
            ]
            is False
        ),

    "result_labels_not_accessed":
        (
            result[
                "labels_accessed"
            ]
            is False
        ),

    "result_gpu_false":
        (
            result[
                "gpu_used"
            ]
            is False
        ),

    "receipt_status":
        (
            receipt[
                "status"
            ]
            ==
            "PASS_EXACT_STAGE20_RECONSTRUCTION"
        ),

    "receipt_all_counts_exact":
        (
            receipt[
                "all_frozen_counts_exact"
            ]
            is True
        ),

    "receipt_mismatch_count_zero":
        (
            len(
                receipt[
                    "mismatches"
                ]
            )
            ==
            0
        ),

    "receipt_timing_false":
        (
            receipt[
                "timing_performed"
            ]
            is False
        ),

    "receipt_perf_metrics_false":
        (
            receipt[
                "performance_metrics_generated"
            ]
            is False
        ),

    "receipt_corpus_not_regenerated":
        (
            receipt[
                "release_corpus_regenerated"
            ]
            is False
        ),

    "receipt_timed_extraction_allowed":
        (
            receipt[
                "timed_extraction_allowed_next"
            ]
            is True
        ),
}


for name, passed in checks.items():

    print(
        f"{name:42s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

    if not passed:

        raise RuntimeError(
            f"Scientific content gate failed: {name}"
        )


# =============================================================================
# 6. EXACT COUNT RECHECK
# =============================================================================

banner(
    "STAGE26-4B2-GIT :: EXACT COUNT RECHECK"
)


actual_counts = result[
    "counts"
]


for metric, expected in EXPECTED_COUNTS.items():

    if metric not in actual_counts:

        raise RuntimeError(
            f"Missing validation metric: {metric}"
        )

    actual = int(
        actual_counts[
            metric
        ]
    )

    passed = (
        actual == expected
    )

    print(
        f"{metric:42s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual:>12,d}"
    )

    if not passed:

        raise RuntimeError(
            f"Historical anchor changed: {metric}"
        )


recomputed_count_fingerprint = hashlib.sha256(
    json.dumps(
        actual_counts,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


print(
    "\nRecorded count fingerprint:"
)

print(
    " ",
    EXPECTED_COUNT_FINGERPRINT
)

print(
    "Recomputed:"
)

print(
    " ",
    recomputed_count_fingerprint
)


if (
    recomputed_count_fingerprint
    !=
    EXPECTED_COUNT_FINGERPRINT
):

    raise RuntimeError(
        "Count fingerprint changed."
    )


# =============================================================================
# 7. BUILD DURABLE VALIDATION CHECKPOINT
# =============================================================================

banner(
    "STAGE26-4B2-GIT :: BUILD DURABLE CHECKPOINT"
)


if CHECKPOINT_DIR.exists():

    raise RuntimeError(
        "Durable Stage26-4B2 checkpoint already exists; "
        "do not overwrite it."
    )


CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


copy_map = {
    SOURCE_CONFIG:
        CHECKPOINT_DIR
        /
        SOURCE_CONFIG.name,

    SOURCE_RESULT:
        CHECKPOINT_DIR
        /
        SOURCE_RESULT.name,

    SOURCE_RECEIPT:
        CHECKPOINT_DIR
        /
        SOURCE_RECEIPT.name,
}


for source, destination in copy_map.items():

    shutil.copy2(
        source,
        destination,
    )


# =============================================================================
# 8. WRITE VALIDATION MANIFEST
# =============================================================================

rows = []


for path in sorted(
    CHECKPOINT_DIR.iterdir()
):

    if (
        path.is_file()
        and
        path != MANIFEST
    ):

        rows.append(
            {
                "repo_relative_path":
                    str(
                        path.relative_to(
                            REPO
                        )
                    ),

                "size_bytes":
                    int(
                        path.stat().st_size
                    ),

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )


manifest = {
    "schema":
        "stage26_4b2_full_monday_validation_manifest_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4B2",

    "status":
        "PASS_EXACT_STAGE20_RECONSTRUCTION_READY_FOR_GIT_ANCHOR",

    "parent_commit":
        EXPECTED_PARENT,

    "protocol_parent":
        EXPECTED_PROTOCOL_PARENT,

    "protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "worker_sha256":
        EXPECTED_WORKER_SHA256,

    "source": {
        "filename":
            "Monday-WorkingHours.pcap",

        "size_bytes":
            EXPECTED_PCAP_SIZE,

        "sha256":
            EXPECTED_PCAP_SHA256,
    },

    "validation": {
        "full_raw_packet_count":
            11_709_971,

        "historical_anchor_count":
            len(
                EXPECTED_COUNTS
            ),

        "matched_anchor_count":
            len(
                EXPECTED_COUNTS
            ),

        "mismatch_count":
            0,

        "count_summary_sha256":
            EXPECTED_COUNT_FINGERPRINT,

        "config_sha256":
            EXPECTED_CONFIG_SHA256,

        "result_sha256":
            EXPECTED_RESULT_SHA256,

        "receipt_sha256":
            EXPECTED_RECEIPT_SHA256,

        "timing_performed":
            False,

        "throughput_measured":
            False,
    },

    "corpus_policy": {
        "release_corpus_authoritative":
            True,

        "release_corpus_regenerated":
            False,

        "new_corpus_created":
            False,
    },

    "scientific_gate": {
        "full_validation_passed":
            True,

        "timed_extraction_allowed_after_anchor":
            True,

        "timed_extraction_performed":
            False,

        "models_loaded":
            False,

        "labels_accessed":
            False,

        "gpu_used":
            False,
    },

    "files":
        rows,
}


atomic_json(
    MANIFEST,
    manifest,
)


manifest_sha = sha256_file(
    MANIFEST
)


print(
    "Manifest:"
)

print(
    " ",
    MANIFEST
)

print(
    "SHA256:"
)

print(
    " ",
    manifest_sha
)


# =============================================================================
# 9. REPOSITORY CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-4B2-GIT :: REPOSITORY CHANGE AUDIT"
)


status_after_build = git(
    "status",
    "--porcelain",
)


print(
    status_after_build
)


if not status_after_build:

    raise RuntimeError(
        "Expected uncommitted validation checkpoint."
    )


unexpected = []


for line in status_after_build.splitlines():

    rel = line[
        3:
    ]

    if not rel.startswith(
        str(
            CHECKPOINT_REL
        )
        +
        "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected Git modification outside Stage26-4B2:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 10. STAGE + COMMIT
# =============================================================================

banner(
    "STAGE26-4B2-GIT :: COMMIT"
)


git(
    "add",
    str(
        CHECKPOINT_REL
    ),
)


staged = git(
    "diff",
    "--cached",
    "--name-status",
)


print(
    staged
)


if not staged:

    raise RuntimeError(
        "Nothing staged for Stage26-4B2."
    )


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-4B2 commit parent mismatch."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Stage26-4B2 commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository is not clean after Stage26-4B2 commit."
    )


# =============================================================================
# 11. PUSH USING KAGGLE SECRET
# =============================================================================

banner(
    "STAGE26-4B2-GIT :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle secret GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    p = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
        check=True,
        text=True,
    )


    print(
        p.stdout.strip()
    )


github_token = None


# =============================================================================
# 12. FETCH + REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-4B2-GIT :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed unexpectedly."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "Remote main does not equal Stage26-4B2 commit."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote Stage26-4B2 parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote Stage26-4B2 subject mismatch."
    )


# =============================================================================
# 13. REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-4B2-GIT :: REMOTE BYTE VERIFICATION"
)


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    str(
        MANIFEST.relative_to(
            REPO
        )
    ),
)

remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "Remote manifest SHA256:"
)

print(
    " ",
    remote_manifest_sha
)


if remote_manifest_sha != manifest_sha:

    raise RuntimeError(
        "Remote validation manifest hash mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


for row in remote_manifest[
    "files"
]:

    rel = row[
        "repo_relative_path"
    ]

    data = git_blob_bytes(
        "origin/main",
        rel,
    )

    actual_size = len(
        data
    )

    actual_sha = sha256_bytes(
        data
    )


    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{rel}"
    )


    if not passed:

        raise RuntimeError(
            f"Remote byte verification failed: {rel}"
        )


print(
    "\n[PASS] Every validation artifact is byte-identical on origin/main."
)


# =============================================================================
# 14. REMOTE SCIENTIFIC CONTENT AUDIT
# =============================================================================

banner(
    "STAGE26-4B2-GIT :: REMOTE SCIENTIFIC CONTENT AUDIT"
)


remote_result_rel = (
    CHECKPOINT_REL
    / SOURCE_RESULT.name
)

remote_receipt_rel = (
    CHECKPOINT_REL
    / SOURCE_RECEIPT.name
)


remote_result = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            remote_result_rel
        ),
    ).decode(
        "utf-8"
    )
)


remote_receipt = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            remote_receipt_rel
        ),
    ).decode(
        "utf-8"
    )
)


remote_checks = {
    "exact_reconstruction":
        (
            remote_receipt[
                "status"
            ]
            ==
            "PASS_EXACT_STAGE20_RECONSTRUCTION"
        ),

    "all_frozen_counts_exact":
        (
            remote_receipt[
                "all_frozen_counts_exact"
            ]
            is True
        ),

    "zero_mismatches":
        (
            len(
                remote_receipt[
                    "mismatches"
                ]
            )
            ==
            0
        ),

    "timing_not_performed":
        (
            remote_result[
                "timing_performed"
            ]
            is False
            and
            remote_receipt[
                "timing_performed"
            ]
            is False
        ),

    "count_fingerprint":
        (
            remote_result[
                "count_summary_sha256"
            ]
            ==
            EXPECTED_COUNT_FINGERPRINT
        ),

    "corpus_not_regenerated":
        (
            remote_receipt[
                "release_corpus_regenerated"
            ]
            is False
        ),

    "gpu_not_used":
        (
            remote_receipt[
                "gpu_used"
            ]
            is False
        ),

    "timed_extraction_now_allowed":
        (
            remote_receipt[
                "timed_extraction_allowed_next"
            ]
            is True
        ),
}


for name, passed in remote_checks.items():

    print(
        f"{name:38s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

    if not passed:

        raise RuntimeError(
            f"Remote scientific validation failed: {name}"
        )


# =============================================================================
# 15. FINAL REPOSITORY AUDIT
# =============================================================================

banner(
    "STAGE26-4B2-GIT :: FINAL AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != final_remote:

    raise RuntimeError(
        "Local/remote divergence after Stage26-4B2."
    )


if final_status:

    raise RuntimeError(
        "Repository not clean after Stage26-4B2 anchor."
    )


# =============================================================================
# 16. CLOSURE
# =============================================================================

banner(
    "STAGE26-4B2-GIT COMPLETE"
)


print(
    "COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nSUBJECT:"
)

print(
    " ",
    COMMIT_SUBJECT
)


print(
    "\nPARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nVALIDATION:"
)

print(
    "  historical anchors : 14"
)

print(
    "  exact matches      : 14"
)

print(
    "  mismatches         : 0"
)

print(
    "  count fingerprint  :",
    EXPECTED_COUNT_FINGERPRINT
)


print(
    "\nFROZEN VALIDATION HASHES:"
)

print(
    "  config  :",
    EXPECTED_CONFIG_SHA256
)

print(
    "  result  :",
    EXPECTED_RESULT_SHA256
)

print(
    "  receipt :",
    EXPECTED_RECEIPT_SHA256
)

print(
    "  manifest:",
    manifest_sha
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  local == origin/main        : PASS"
)

print(
    "  exact parent                : PASS"
)

print(
    "  every validation file       : PASS"
)

print(
    "  scientific content          : PASS"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  SOURCE-FAITHFUL EXTRACTOR VALIDATED : YES"
)

print(
    "  FULL MONDAY EXACT MATCH             : YES"
)

print(
    "  EXTRACTION TIMING PERFORMED         : NO"
)

print(
    "  EXTRACTION THROUGHPUT MEASURED      : NO"
)

print(
    "  RELEASE CORPUS REGENERATED          : NO"
)

print(
    "  MODELS LOADED                       : NO"
)

print(
    "  GPU                                 : NO"
)


print(
    "\nNEXT:"
)

print(
    "  STAGE26-4B3 — execute the five already-frozen"
)

print(
    "  1,000,000-packet CPU1 extraction repetitions."
)

print(
    "  This will be the FIRST extraction performance measurement."
)


STAGE26-4B2-GIT :: PRE-ANCHOR GIT GATE
Expected parent: b41a0559598a70e6c69aac05c76e4c4edb73b34e
Local HEAD     : b41a0559598a70e6c69aac05c76e4c4edb73b34e
origin/main    : b41a0559598a70e6c69aac05c76e4c4edb73b34e
Repo clean     : True

STAGE26-4B2-GIT :: VALIDATION OUTPUT IDENTITY
config       PASS 409f43dfe167b17595b101b37b92d0796c757fd7cdb8c0946d9969f3c61ce122
result       PASS a012ba81b4cbb339371898e6adc4cd43a1c6364086d1b343f0fb7cd73c3bb46f
receipt      PASS 0634cd71d3b1d2111a7b44a3519711660fb6ec3ad8f143804ba1cd3b9e6802d2

STAGE26-4B2-GIT :: SCIENTIFIC CONTENT GATE
config_mode_untimed                       : PASS
config_protocol_commit                    : PASS
config_protocol_sha                       : PASS
config_worker_sha                         : PASS
config_source_sha                         : PASS
result_status_pass                        : PASS
result_mode_untimed                       : PASS
result_timing_false                       : PASS
result_count_fingerprint        

In [17]:
# =============================================================================
# STAGE26-4B3
# FIRST RAW EXTRACTION PERFORMANCE MEASUREMENT
#
# DURABLE SCIENTIFIC PARENT:
#   e558da134162652c1a955f60f15c175f083923c8
#
# PRECONDITIONS ALREADY SATISFIED:
#   - extraction protocol remotely frozen;
#   - source-faithful worker remotely frozen;
#   - full 11,709,971-packet Monday correctness pass completed;
#   - 14 / 14 historical Stage20 anchors exact;
#   - correctness result remotely anchored.
#
# FROZEN MEASUREMENT:
#   source        : Monday-WorkingHours.pcap
#   sample        : raw packet records 1 .. 1,000,000
#   CPU           : affinity [0], thread_count 1
#   repetitions   : 5
#   fresh process : YES
#   timeout/rep   : 600 s
#
# TIMER BOUNDARY INSIDE FROZEN WORKER:
#   START immediately before opening/reading PCAP
#   STOP  after in-memory flow reconstruction + declared sample EOF
#
# INCLUDED:
#   - application disk/file read
#   - PCAPNG decoding
#   - Ethernet/VLAN -> IPv4 parsing
#   - TCP/UDP/OTHER0 normalization
#   - bidirectional flow keying
#   - timeout / FIN / sample-EOF lifecycle
#
# EXCLUDED:
#   - Python process startup
#   - JSON result serialization
#   - JVM startup
#   - corpus creation
#   - release-corpus reconstruction
#   - models / inference
#   - GPU
#
# PAGE CACHE:
#   No cache flushing/manipulation.
#   All five repetitions are retained as frozen.
#
# THIS CELL:
#   - performs the first Stage26 extraction timing;
#   - writes outputs OUTSIDE Git;
#   - does NOT commit/push.
# =============================================================================

from __future__ import annotations

import os
import sys
import csv
import json
import time
import signal
import hashlib
import statistics
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import psutil


# =============================================================================
# 0. FROZEN IDENTITIES / PATHS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

EXTRACTION_ROOT = (
    STAGE26_ROOT
    / "extraction"
)

RUN_ROOT = (
    EXTRACTION_ROOT
    / "stage26_4b3_cpu1_extraction_timing"
)


EXPECTED_HEAD = (
    "e558da134162652c1a955f60f15c175f083923c8"
)

EXPECTED_PROTOCOL_COMMIT = (
    "b41a0559598a70e6c69aac05c76e4c4edb73b34e"
)


LOCK_ROOT = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4b_extraction_protocol_lock"
)

VALIDATION_ROOT = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4b2_full_monday_validation"
)


WORKER = (
    LOCK_ROOT
    / "stage26_raw_flow_extractor_v1.py"
)

PROTOCOL = (
    LOCK_ROOT
    / "stage26_extraction_protocol.json"
)

PROTOCOL_MANIFEST = (
    LOCK_ROOT
    / "stage26_4b_extraction_lock_manifest.json"
)


VALIDATION_RESULT = (
    VALIDATION_ROOT
    / "stage26_4b2_full_monday_validation_result.json"
)

VALIDATION_RECEIPT = (
    VALIDATION_ROOT
    / "stage26_4b2_full_monday_validation_receipt.json"
)

VALIDATION_MANIFEST = (
    VALIDATION_ROOT
    / "stage26_4b2_validation_manifest.json"
)


EXPECTED_WORKER_SHA256 = (
    "ba34c5ab1a8e0bdb22acf5d6be3430cc2b31fdec04dc333dcf3ef7344abe44e6"
)

EXPECTED_PROTOCOL_SHA256 = (
    "12f7f8f0332d92694a827e3473dc1cef66939cf1cf1c49c17ad1699e60d929b0"
)

EXPECTED_PROTOCOL_MANIFEST_SHA256 = (
    "7fe04d2628e1bfe7da5551192f45821b5406b3b3087976c4334bcd74224fc2d3"
)

EXPECTED_VALIDATION_RESULT_SHA256 = (
    "a012ba81b4cbb339371898e6adc4cd43a1c6364086d1b343f0fb7cd73c3bb46f"
)

EXPECTED_VALIDATION_RECEIPT_SHA256 = (
    "0634cd71d3b1d2111a7b44a3519711660fb6ec3ad8f143804ba1cd3b9e6802d2"
)

EXPECTED_VALIDATION_MANIFEST_SHA256 = (
    "87cc9f272e451a6ea032a1aa12f147bea311f80eadce0cfb51ef8f734efccd32"
)

EXPECTED_FULL_COUNT_FINGERPRINT = (
    "2ac0f791b97187cb10a4228775146958830719b29dc10a064ed5adae0614728e"
)


MONDAY_PCAP = (
    STAGE26_ROOT
    / "sources"
    / "Monday-WorkingHours.pcap"
)

EXPECTED_PCAP_SIZE = (
    10_822_507_416
)

EXPECTED_PCAP_SHA256 = (
    "f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972"
)


# =============================================================================
# 1. FROZEN EXECUTION PARAMETERS
# =============================================================================

PACKET_LIMIT = (
    1_000_000
)

REPETITIONS = (
    5
)

AFFINITY = [
    0
]

THREAD_COUNT = (
    1
)

TIMEOUT_SECONDS = (
    600
)


# Environment gate frozen in Stage26 protocol.
MAX_CPU_UTIL_PERCENT = (
    20.0
)

MIN_AVAILABLE_RAM_GIB = (
    8.0
)

ENV_GATE_ATTEMPTS = (
    3
)

ENV_GATE_RETRY_SECONDS = (
    5
)

CPU_SAMPLE_SECONDS = (
    1.0
)


# =============================================================================
# 2. OUTPUTS
# =============================================================================

SUMMARY_JSON = (
    RUN_ROOT
    / "stage26_4b3_cpu1_extraction_summary.json"
)

SUMMARY_CSV = (
    RUN_ROOT
    / "stage26_4b3_cpu1_extraction_repetitions.csv"
)

RECEIPT_JSON = (
    RUN_ROOT
    / "stage26_4b3_cpu1_extraction_receipt.json"
)


# =============================================================================
# 3. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 122
    )

    print(text)

    print(
        "=" * 122
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            p.stdout
        )

    return p


def git(*args):

    return run(
        [
            "git",
            *args,
        ]
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:

                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def median(values):

    return float(
        statistics.median(
            values
        )
    )


def env_gate():

    attempts = []


    for attempt in range(
        1,
        ENV_GATE_ATTEMPTS + 1,
    ):

        # Sample system utilization for the exact frozen interval.
        cpu_util = float(
            psutil.cpu_percent(
                interval=CPU_SAMPLE_SECONDS
            )
        )


        vm = psutil.virtual_memory()

        available_gib = (
            float(
                vm.available
            )
            /
            1024**3
        )


        cpu_pass = (
            cpu_util
            <=
            MAX_CPU_UTIL_PERCENT
        )

        ram_pass = (
            available_gib
            >=
            MIN_AVAILABLE_RAM_GIB
        )


        attempt_record = {
            "attempt":
                attempt,

            "cpu_utilization_percent":
                cpu_util,

            "available_ram_gib":
                available_gib,

            "cpu_gate_pass":
                cpu_pass,

            "ram_gate_pass":
                ram_pass,

            "gate_pass":
                (
                    cpu_pass
                    and
                    ram_pass
                ),
        }


        attempts.append(
            attempt_record
        )


        print(
            f"  gate attempt {attempt}/{ENV_GATE_ATTEMPTS}: "
            f"CPU={cpu_util:.1f}% "
            f"(<= {MAX_CPU_UTIL_PERCENT:.1f}%) | "
            f"RAM={available_gib:.3f} GiB "
            f"(>= {MIN_AVAILABLE_RAM_GIB:.1f}) | "
            f"{'PASS' if attempt_record['gate_pass'] else 'RETRY'}"
        )


        if attempt_record[
            "gate_pass"
        ]:

            return (
                True,
                attempts,
            )


        if attempt < ENV_GATE_ATTEMPTS:

            time.sleep(
                ENV_GATE_RETRY_SECONDS
            )


    return (
        False,
        attempts,
    )


# =============================================================================
# 4. DURABLE SCIENTIFIC GATE
# =============================================================================

banner(
    "STAGE26-4B3 :: DURABLE SCIENTIFIC GATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD :",
    EXPECTED_HEAD
)

print(
    "Local HEAD    :",
    head
)

print(
    "origin/main   :",
    remote
)

print(
    "Repo clean    :",
    status == ""
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected durable Stage26 parent."
    )


if remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed before Stage26-4B3."
    )


if status:

    raise RuntimeError(
        "Repository must be clean before timed extraction."
    )


# =============================================================================
# 5. FROZEN PROTOCOL / VALIDATION HASH GATE
# =============================================================================

banner(
    "STAGE26-4B3 :: FROZEN PROVENANCE HASH GATE"
)


hash_checks = [
    (
        "worker",
        WORKER,
        EXPECTED_WORKER_SHA256,
    ),
    (
        "protocol",
        PROTOCOL,
        EXPECTED_PROTOCOL_SHA256,
    ),
    (
        "protocol manifest",
        PROTOCOL_MANIFEST,
        EXPECTED_PROTOCOL_MANIFEST_SHA256,
    ),
    (
        "validation result",
        VALIDATION_RESULT,
        EXPECTED_VALIDATION_RESULT_SHA256,
    ),
    (
        "validation receipt",
        VALIDATION_RECEIPT,
        EXPECTED_VALIDATION_RECEIPT_SHA256,
    ),
    (
        "validation manifest",
        VALIDATION_MANIFEST,
        EXPECTED_VALIDATION_MANIFEST_SHA256,
    ),
]


for name, path, expected in hash_checks:

    if not path.exists():

        raise FileNotFoundError(
            path
        )


    actual = sha256_file(
        path
    )


    passed = (
        actual
        ==
        expected
    )


    print(
        f"{name:28s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Frozen provenance changed: {name}"
        )


# =============================================================================
# 6. VALIDATION PERMISSION GATE
# =============================================================================

banner(
    "STAGE26-4B3 :: TIMING PERMISSION GATE"
)


validation_result = json.loads(
    VALIDATION_RESULT.read_text(
        encoding="utf-8"
    )
)

validation_receipt = json.loads(
    VALIDATION_RECEIPT.read_text(
        encoding="utf-8"
    )
)

protocol = json.loads(
    PROTOCOL.read_text(
        encoding="utf-8"
    )
)


permission_checks = {
    "full_validation_exact":
        (
            validation_receipt[
                "status"
            ]
            ==
            "PASS_EXACT_STAGE20_RECONSTRUCTION"
        ),

    "all_frozen_counts_exact":
        (
            validation_receipt[
                "all_frozen_counts_exact"
            ]
            is True
        ),

    "zero_validation_mismatches":
        (
            len(
                validation_receipt[
                    "mismatches"
                ]
            )
            ==
            0
        ),

    "full_count_fingerprint":
        (
            validation_result[
                "count_summary_sha256"
            ]
            ==
            EXPECTED_FULL_COUNT_FINGERPRINT
        ),

    "timing_allowed_next":
        (
            validation_receipt[
                "timed_extraction_allowed_next"
            ]
            is True
        ),

    "prior_timing_false":
        (
            validation_receipt[
                "timing_performed"
            ]
            is False
        ),

    "corpus_not_regenerated":
        (
            validation_receipt[
                "release_corpus_regenerated"
            ]
            is False
        ),

    "gpu_not_used":
        (
            validation_receipt[
                "gpu_used"
            ]
            is False
        ),
}


for name, passed in permission_checks.items():

    print(
        f"{name:38s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    permission_checks.values()
):

    raise RuntimeError(
        "Timed extraction permission gate failed."
    )


# Verify notebook parameters equal frozen protocol.
frozen_sample = protocol[
    "timed_sample"
]

frozen_execution = protocol[
    "execution"
]


parameter_checks = {
    "packet_limit":
        (
            frozen_sample[
                "raw_packet_count"
            ]
            ==
            PACKET_LIMIT
        ),

    "packet_start":
        (
            frozen_sample[
                "raw_packet_start_index"
            ]
            ==
            1
        ),

    "packet_end":
        (
            frozen_sample[
                "raw_packet_end_index"
            ]
            ==
            PACKET_LIMIT
        ),

    "repetitions":
        (
            frozen_execution[
                "repetitions"
            ]
            ==
            REPETITIONS
        ),

    "affinity":
        (
            frozen_execution[
                "affinity"
            ]
            ==
            AFFINITY
        ),

    "thread_count":
        (
            frozen_execution[
                "thread_count"
            ]
            ==
            THREAD_COUNT
        ),

    "timeout":
        (
            frozen_execution[
                "timeout_seconds_per_repetition"
            ]
            ==
            TIMEOUT_SECONDS
        ),
}


print(
    "\nFrozen execution parameters:"
)


for name, passed in parameter_checks.items():

    print(
        f"{name:38s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    parameter_checks.values()
):

    raise RuntimeError(
        "Notebook measurement parameters differ from frozen protocol."
    )


# =============================================================================
# 7. SOURCE PRESENCE / IDENTITY GATE
# =============================================================================

banner(
    "STAGE26-4B3 :: SOURCE GATE"
)


if not MONDAY_PCAP.exists():

    raise FileNotFoundError(
        MONDAY_PCAP
    )


resolved_source = MONDAY_PCAP.resolve(
    strict=True
)

source_size = int(
    resolved_source.stat().st_size
)


print(
    "Canonical source:"
)

print(
    " ",
    MONDAY_PCAP
)

print(
    "Resolved source:"
)

print(
    " ",
    resolved_source
)

print(
    "Size bytes:"
)

print(
    " ",
    source_size
)

print(
    "Previously verified SHA256:"
)

print(
    " ",
    EXPECTED_PCAP_SHA256
)


if source_size != EXPECTED_PCAP_SIZE:

    raise RuntimeError(
        "Monday source size changed."
    )


print(
    "\nFull 10.08-GiB SHA is NOT redundantly recomputed."
)


# =============================================================================
# 8. CPU / RUNTIME GATE
# =============================================================================

banner(
    "STAGE26-4B3 :: CPU EXECUTION GATE"
)


if not hasattr(
    os,
    "sched_getaffinity",
):

    raise RuntimeError(
        "CPU affinity API unavailable."
    )


available_affinity = sorted(
    os.sched_getaffinity(
        0
    )
)


print(
    "Current allowed CPUs:",
    available_affinity
)

print(
    "Required worker CPU :",
    AFFINITY
)


if not set(
    AFFINITY
).issubset(
    set(
        available_affinity
    )
):

    raise RuntimeError(
        "Frozen CPU affinity is not available in this runtime."
    )


print(
    "Physical CPU cores:",
    psutil.cpu_count(
        logical=False
    )
)

print(
    "Logical CPU cores :",
    psutil.cpu_count(
        logical=True
    )
)

print(
    "GPU visible       :",
    bool(
        os.environ.get(
            "CUDA_VISIBLE_DEVICES",
            ""
        ).strip()
    )
)


# =============================================================================
# 9. FIRST-MEASUREMENT NON-OVERWRITE GATE
# =============================================================================

banner(
    "STAGE26-4B3 :: NON-OVERWRITE GATE"
)


if RUN_ROOT.exists():

    existing = list(
        RUN_ROOT.rglob(
            "*"
        )
    )

    if existing:

        print(
            "Existing Stage26-4B3 files:"
        )

        for p in existing:

            print(
                " ",
                p
            )

        raise RuntimeError(
            "Stage26-4B3 outputs already exist. "
            "Do NOT overwrite or remeasure."
        )


RUN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Fresh Stage26-4B3 output directory: PASS"
)


# =============================================================================
# 10. WORKER ENVIRONMENT
# =============================================================================

worker_env = os.environ.copy()


# Explicit CPU-only context.
worker_env[
    "CUDA_VISIBLE_DEVICES"
] = ""


worker_env[
    "PYTHONHASHSEED"
] = "0"


for env_name in [
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
]:

    worker_env[
        env_name
    ] = "1"


# =============================================================================
# 11. EXECUTE FIVE FROZEN REPETITIONS
# =============================================================================

banner(
    "STAGE26-4B3 :: FIRST EXTRACTION PERFORMANCE MEASUREMENT"
)


print(
    "Frozen packet prefix : 1 .. 1,000,000"
)

print(
    "Fresh processes      : 5"
)

print(
    "CPU affinity         : [0]"
)

print(
    "Timeout / repetition : 600 s"
)

print(
    "Disk I/O included    : YES"
)

print(
    "Serialization        : EXCLUDED"
)

print(
    "Python startup       : EXCLUDED"
)

print(
    "Cache manipulation   : NONE"
)

print(
    "\nStarting measurements..."
)


rep_records = []

count_dicts = []


for rep_index in range(
    1,
    REPETITIONS + 1,
):

    banner(
        f"STAGE26-4B3 :: REPETITION {rep_index}/{REPETITIONS}"
    )


    # -------------------------------------------------------------------------
    # Frozen environment gate
    # -------------------------------------------------------------------------

    gate_passed, gate_attempts = env_gate()


    if not gate_passed:

        failure_record = {
            "repetition":
                rep_index,

            "status":
                "INVALID_ENVIRONMENT",

            "environment_gate":
                gate_attempts,

            "measurement_performed":
                False,
        }


        rep_records.append(
            failure_record
        )


        failure_path = (
            RUN_ROOT
            / f"rep_{rep_index:02d}_invalid_environment.json"
        )


        atomic_json(
            failure_path,
            failure_record,
        )


        print(
            "\n[STOP] Environment gate failed all frozen attempts."
        )

        print(
            "No timing was performed for this repetition."
        )

        break


    # -------------------------------------------------------------------------
    # Write per-repetition worker config
    # -------------------------------------------------------------------------

    config_path = (
        RUN_ROOT
        / f"rep_{rep_index:02d}_config.json"
    )

    result_path = (
        RUN_ROOT
        / f"rep_{rep_index:02d}_result.json"
    )


    config = {
        "schema":
            "stage26_4b3_extraction_timing_config_v1",

        "condition":
            "CPU1_RAW_PACKET_PREFIX_1000000",

        "repetition":
            rep_index,

        "mode":
            "BENCHMARK_TIMED",

        "pcap_path":
            str(
                MONDAY_PCAP
            ),

        "result_path":
            str(
                result_path
            ),

        "affinity":
            AFFINITY,

        "packet_limit":
            PACKET_LIMIT,

        "protocol_commit":
            EXPECTED_PROTOCOL_COMMIT,

        "durable_validation_commit":
            EXPECTED_HEAD,

        "protocol_sha256":
            EXPECTED_PROTOCOL_SHA256,

        "worker_sha256":
            EXPECTED_WORKER_SHA256,

        "source_sha256":
            EXPECTED_PCAP_SHA256,

        "environment_gate":
            gate_attempts,

        "corpus_creation_allowed":
            False,

        "gpu_allowed":
            False,
    }


    atomic_json(
        config_path,
        config,
    )


    config_sha = sha256_file(
        config_path
    )


    print(
        "\nConfig SHA256:"
    )

    print(
        " ",
        config_sha
    )


    print(
        "\nLaunching fresh frozen worker...",
        flush=True,
    )


    # -------------------------------------------------------------------------
    # Exact frozen 600-second resource timeout.
    #
    # IMPORTANT:
    # Parent does not measure performance.
    # Worker owns the performance timer.
    # -------------------------------------------------------------------------

    try:

        process = subprocess.run(
            [
                sys.executable,
                str(
                    WORKER
                ),
                str(
                    config_path
                ),
            ],
            cwd=REPO,
            env=worker_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            check=False,
            timeout=TIMEOUT_SECONDS,
        )


    except subprocess.TimeoutExpired as exc:

        timeout_record = {
            "repetition":
                rep_index,

            "status":
                "TIMEOUT_RESOURCE_LIMIT",

            "timeout_seconds":
                TIMEOUT_SECONDS,

            "environment_gate":
                gate_attempts,

            "measurement_completed":
                False,

            "protocol_adapted":
                False,
        }


        rep_records.append(
            timeout_record
        )


        timeout_path = (
            RUN_ROOT
            / f"rep_{rep_index:02d}_timeout.json"
        )


        atomic_json(
            timeout_path,
            timeout_record,
        )


        print(
            "\n[TIMEOUT_RESOURCE_LIMIT]"
        )

        print(
            "Frozen 600-second limit reached."
        )

        print(
            "No sample/timeout/iteration adaptation performed."
        )

        continue


    print(
        "Worker return code:",
        process.returncode
    )


    if process.stdout.strip():

        print(
            "\nWorker output:"
        )

        print(
            process.stdout
        )


    # -------------------------------------------------------------------------
    # Non-zero worker exit
    # -------------------------------------------------------------------------

    if process.returncode != 0:

        # SIGKILL is commonly the kernel/cgroup OOM path.
        if process.returncode in (
            -signal.SIGKILL,
            137,
        ):

            status_name = (
                "RESOURCE_LIMIT_OOM"
            )


        else:

            status_name = (
                "IMPLEMENTATION_OR_RUNTIME_FAILURE"
            )


        failure_record = {
            "repetition":
                rep_index,

            "status":
                status_name,

            "return_code":
                process.returncode,

            "stdout":
                process.stdout,

            "environment_gate":
                gate_attempts,

            "measurement_completed":
                False,

            "protocol_adapted":
                False,
        }


        rep_records.append(
            failure_record
        )


        failure_path = (
            RUN_ROOT
            / f"rep_{rep_index:02d}_failure.json"
        )


        atomic_json(
            failure_path,
            failure_record,
        )


        print(
            f"\n[{status_name}]"
        )


        if status_name == "IMPLEMENTATION_OR_RUNTIME_FAILURE":

            print(
                "Stopping for scoped diagnosis."
            )

            break


        continue


    # -------------------------------------------------------------------------
    # Successful result gate
    # -------------------------------------------------------------------------

    if not result_path.exists():

        raise RuntimeError(
            f"Rep {rep_index}: worker succeeded but result JSON is missing."
        )


    result_sha = sha256_file(
        result_path
    )


    result = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


    result_checks = {
        "status":
            (
                result.get(
                    "status"
                )
                ==
                "PASS"
            ),

        "mode":
            (
                result.get(
                    "mode"
                )
                ==
                "BENCHMARK_TIMED"
            ),

        "timing":
            (
                result.get(
                    "timing_performed"
                )
                is True
            ),

        "packet_limit":
            (
                result.get(
                    "packet_limit"
                )
                ==
                PACKET_LIMIT
            ),

        "raw_packets":
            (
                result[
                    "counts"
                ][
                    "raw_packet_count"
                ]
                ==
                PACKET_LIMIT
            ),

        "disk_IO":
            (
                result.get(
                    "disk_IO_included"
                )
                is True
            ),

        "serialization":
            (
                result.get(
                    "serialization_included"
                )
                is False
            ),

        "Python_startup":
            (
                result.get(
                    "Python_process_startup_included"
                )
                is False
            ),

        "JVM_startup":
            (
                result.get(
                    "JVM_startup_included"
                )
                is False
            ),

        "corpus":
            (
                result.get(
                    "corpus_created_or_modified"
                )
                is False
            ),

        "labels":
            (
                result.get(
                    "labels_accessed"
                )
                is False
            ),

        "gpu":
            (
                result.get(
                    "gpu_used"
                )
                is False
            ),
    }


    if not all(
        result_checks.values()
    ):

        print(
            result_checks
        )

        raise RuntimeError(
            f"Rep {rep_index}: frozen result semantics failed."
        )


    counts = result[
        "counts"
    ]


    # Verify worker's count fingerprint.
    recomputed_count_sha = hashlib.sha256(
        json.dumps(
            counts,
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
        ).encode(
            "utf-8"
        )
    ).hexdigest()


    if (
        recomputed_count_sha
        !=
        result[
            "count_summary_sha256"
        ]
    ):

        raise RuntimeError(
            f"Rep {rep_index}: count fingerprint mismatch."
        )


    count_dicts.append(
        counts
    )


    record = {
        "repetition":
            rep_index,

        "status":
            "PASS",

        "config_path":
            str(
                config_path
            ),

        "config_sha256":
            config_sha,

        "result_path":
            str(
                result_path
            ),

        "result_sha256":
            result_sha,

        "count_summary_sha256":
            result[
                "count_summary_sha256"
            ],

        "environment_gate":
            gate_attempts,

        "elapsed_ns":
            int(
                result[
                    "elapsed_ns"
                ]
            ),

        "elapsed_seconds":
            float(
                result[
                    "elapsed_seconds"
                ]
            ),

        "packets_per_second":
            float(
                result[
                    "packets_per_second"
                ]
            ),

        "bytes_per_second":
            float(
                result[
                    "bytes_per_second"
                ]
            ),

        "MiB_per_second":
            float(
                result[
                    "MiB_per_second"
                ]
            ),

        "flows_per_second":
            float(
                result[
                    "flows_per_second"
                ]
            ),

        "completed_lifecycle_flows_per_second":
            float(
                result[
                    "completed_lifecycle_flows_per_second"
                ]
            ),

        "container_bytes_per_second":
            float(
                result[
                    "container_bytes_per_second"
                ]
            ),

        "captured_ipv4_bytes_per_second":
            float(
                result[
                    "captured_ipv4_bytes_per_second"
                ]
            ),

        "raw_packet_count":
            int(
                counts[
                    "raw_packet_count"
                ]
            ),

        "valid_ipv4_packet_count":
            int(
                counts[
                    "valid_ipv4_packet_count"
                ]
            ),

        "exportable_flow_count":
            int(
                counts[
                    "exportable_flow_count"
                ]
            ),

        "finished_exportable_flows":
            int(
                counts[
                    "finished_exportable_flows"
                ]
            ),

        "eof_current_exportable_flows":
            int(
                counts[
                    "eof_current_exportable_flows"
                ]
            ),

        "max_active_flows":
            int(
                counts[
                    "max_active_flows"
                ]
            ),

        "pcap_file_bytes_consumed":
            int(
                counts[
                    "pcap_file_bytes_consumed"
                ]
            ),
    }


    rep_records.append(
        record
    )


    print(
        "\nPASS"
    )

    print(
        f"  elapsed             : {record['elapsed_seconds']:.6f} s"
    )

    print(
        f"  packets/s           : {record['packets_per_second']:,.2f}"
    )

    print(
        f"  captured MiB/s      : {record['MiB_per_second']:,.2f}"
    )

    print(
        f"  flows/s             : {record['flows_per_second']:,.2f}"
    )

    print(
        f"  exportable flows    : {record['exportable_flow_count']:,}"
    )

    print(
        f"  count fingerprint   : {record['count_summary_sha256']}"
    )


# =============================================================================
# 12. EXECUTION COMPLETENESS GATE
# =============================================================================

banner(
    "STAGE26-4B3 :: EXECUTION COMPLETENESS"
)


pass_records = [
    r
    for r in rep_records
    if r[
        "status"
    ]
    ==
    "PASS"
]


status_counts = {}


for row in rep_records:

    status_counts[
        row[
            "status"
        ]
    ] = (
        status_counts.get(
            row[
                "status"
            ],
            0,
        )
        +
        1
    )


print(
    "Requested repetitions:",
    REPETITIONS
)

print(
    "Records produced      :",
    len(
        rep_records
    )
)

print(
    "PASS                  :",
    len(
        pass_records
    )
)


for status_name, count in sorted(
    status_counts.items()
):

    print(
        f"  {status_name:36s}: {count}"
    )


# =============================================================================
# 13. DETERMINISTIC PREFIX COUNT CONSISTENCY
# =============================================================================

banner(
    "STAGE26-4B3 :: DETERMINISTIC COUNT CONSISTENCY"
)


count_consistency_pass = (
    len(
        count_dicts
    )
    <=
    1
    or
    all(
        c == count_dicts[
            0
        ]
        for c in count_dicts[
            1:
        ]
    )
)


print(
    "Successful repetitions:",
    len(
        count_dicts
    )
)

print(
    "All successful count dictionaries identical:",
    count_consistency_pass
)


if not count_consistency_pass:

    raise RuntimeError(
        "Identical frozen packet prefix produced different flow-count results."
    )


if count_dicts:

    prefix_count_fingerprint = hashlib.sha256(
        json.dumps(
            count_dicts[
                0
            ],
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
        ).encode(
            "utf-8"
        )
    ).hexdigest()


    print(
        "Frozen prefix count fingerprint:"
    )

    print(
        " ",
        prefix_count_fingerprint
    )


else:

    prefix_count_fingerprint = None


# =============================================================================
# 14. FROZEN SUMMARY STATISTICS
# =============================================================================

banner(
    "STAGE26-4B3 :: FROZEN SUMMARY"
)


summary_metrics = {}


metric_names = [
    "elapsed_seconds",
    "packets_per_second",
    "bytes_per_second",
    "MiB_per_second",
    "flows_per_second",
    "completed_lifecycle_flows_per_second",
    "container_bytes_per_second",
    "captured_ipv4_bytes_per_second",
]


if pass_records:

    for metric in metric_names:

        values = [
            float(
                row[
                    metric
                ]
            )
            for row in pass_records
        ]


        summary_metrics[
            metric
        ] = {
            "n":
                len(
                    values
                ),

            "median":
                median(
                    values
                ),

            "minimum":
                float(
                    min(
                        values
                    )
                ),

            "maximum":
                float(
                    max(
                        values
                    )
                ),
        }


        s = summary_metrics[
            metric
        ]


        print(
            f"{metric:42s} "
            f"median={s['median']:,.6f}  "
            f"min={s['minimum']:,.6f}  "
            f"max={s['maximum']:,.6f}"
        )


else:

    print(
        "No successful timed repetitions; no performance summary generated."
    )


# =============================================================================
# 15. WRITE REPETITION CSV
# =============================================================================

csv_columns = [
    "repetition",
    "status",
    "elapsed_seconds",
    "packets_per_second",
    "bytes_per_second",
    "MiB_per_second",
    "flows_per_second",
    "completed_lifecycle_flows_per_second",
    "container_bytes_per_second",
    "captured_ipv4_bytes_per_second",
    "raw_packet_count",
    "valid_ipv4_packet_count",
    "exportable_flow_count",
    "finished_exportable_flows",
    "eof_current_exportable_flows",
    "max_active_flows",
    "pcap_file_bytes_consumed",
    "count_summary_sha256",
    "config_sha256",
    "result_sha256",
]


with SUMMARY_CSV.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=csv_columns,
    )

    writer.writeheader()


    for row in rep_records:

        writer.writerow(
            {
                key:
                    row.get(
                        key
                    )
                for key in csv_columns
            }
        )


csv_sha = sha256_file(
    SUMMARY_CSV
)


# =============================================================================
# 16. WRITE SUMMARY JSON
# =============================================================================

measurement_complete = (
    len(
        rep_records
    )
    ==
    REPETITIONS
    and
    len(
        pass_records
    )
    ==
    REPETITIONS
)


summary = {
    "schema":
        "stage26_4b3_cpu1_extraction_summary_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4B3",

    "scientific_parent":
        EXPECTED_HEAD,

    "protocol_commit":
        EXPECTED_PROTOCOL_COMMIT,

    "protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "worker_sha256":
        EXPECTED_WORKER_SHA256,

    "validation": {
        "commit":
            EXPECTED_HEAD,

        "result_sha256":
            EXPECTED_VALIDATION_RESULT_SHA256,

        "receipt_sha256":
            EXPECTED_VALIDATION_RECEIPT_SHA256,

        "historical_anchors_exact":
            True,

        "historical_anchor_count":
            14,

        "mismatches":
            0,
    },

    "source": {
        "filename":
            "Monday-WorkingHours.pcap",

        "size_bytes":
            EXPECTED_PCAP_SIZE,

        "sha256":
            EXPECTED_PCAP_SHA256,
    },

    "condition": {
        "cpu_mode":
            "CPU_1_PHYSICAL_CORE",

        "affinity":
            AFFINITY,

        "thread_count":
            THREAD_COUNT,

        "packet_sampling_rule":
            "DETERMINISTIC_CONTIGUOUS_RAW_PACKET_PREFIX",

        "packet_start":
            1,

        "packet_end":
            PACKET_LIMIT,

        "packet_count":
            PACKET_LIMIT,

        "fresh_process_per_repetition":
            True,

        "requested_repetitions":
            REPETITIONS,

        "timeout_seconds":
            TIMEOUT_SECONDS,

        "disk_IO_included":
            True,

        "serialization_included":
            False,

        "Python_process_startup_included":
            False,

        "JVM_startup_included":
            False,

        "page_cache_manipulated":
            False,

        "gpu":
            False,
    },

    "status_counts":
        status_counts,

    "measurement_complete":
        measurement_complete,

    "successful_repetitions":
        len(
            pass_records
        ),

    "prefix_count_consistency":
        count_consistency_pass,

    "prefix_count_summary_sha256":
        prefix_count_fingerprint,

    "summary_policy":
        {
            "statistics":
                [
                    "median",
                    "minimum",
                    "maximum",
                ],

            "p95_p99_from_five_repetitions":
                False,
        },

    "metrics":
        summary_metrics,

    "repetitions":
        rep_records,

    "scientific_boundaries": {
        "corpus_created":
            False,

        "release_corpus_regenerated":
            False,

        "labels_accessed":
            False,

        "models_loaded":
            False,

        "Thursday_accessed":
            False,

        "Friday_accessed":
            False,

        "representation_timing":
            False,

        "complete_end_to_end_claim":
            False,

        "gpu_used":
            False,
    },
}


atomic_json(
    SUMMARY_JSON,
    summary,
)


summary_sha = sha256_file(
    SUMMARY_JSON
)


# =============================================================================
# 17. WRITE MEASUREMENT RECEIPT
# =============================================================================

receipt_status = (
    "PASS_FIVE_FROZEN_REPETITIONS"
    if measurement_complete
    else
    "COMPLETE_WITH_FROZEN_RESOURCE_OR_ENVIRONMENT_OUTCOMES"
)


receipt = {
    "schema":
        "stage26_4b3_cpu1_extraction_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4B3",

    "status":
        receipt_status,

    "scientific_parent":
        EXPECTED_HEAD,

    "protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "worker_sha256":
        EXPECTED_WORKER_SHA256,

    "full_monday_correctness_gate":
        "PASS_14_OF_14_EXACT",

    "performance_measurement_started":
        True,

    "performance_measurement_type":
        "RAW_PCAP_TO_FLOW_RECONSTRUCTION_CPU1",

    "requested_repetitions":
        REPETITIONS,

    "successful_repetitions":
        len(
            pass_records
        ),

    "status_counts":
        status_counts,

    "prefix_count_summary_sha256":
        prefix_count_fingerprint,

    "deterministic_count_consistency":
        count_consistency_pass,

    "summary_json":
        str(
            SUMMARY_JSON
        ),

    "summary_json_sha256":
        summary_sha,

    "summary_csv":
        str(
            SUMMARY_CSV
        ),

    "summary_csv_sha256":
        csv_sha,

    "corpus_created":
        False,

    "release_corpus_regenerated":
        False,

    "labels_accessed":
        False,

    "models_loaded":
        False,

    "gpu_used":
        False,

    "complete_end_to_end_claim_allowed":
        False,

    "next_action":
        "GIT_ANCHOR_STAGE26_4B3_BEFORE_FURTHER_PROFILING",
}


atomic_json(
    RECEIPT_JSON,
    receipt,
)


receipt_sha = sha256_file(
    RECEIPT_JSON
)


# =============================================================================
# 18. FINAL GIT AUDIT
# =============================================================================

banner(
    "STAGE26-4B3 :: FINAL GIT AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_HEAD:

    raise RuntimeError(
        "HEAD changed during Stage26-4B3."
    )


if final_remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed during Stage26-4B3."
    )


if final_status:

    print(
        final_status
    )

    raise RuntimeError(
        "Repository changed during Stage26-4B3."
    )


# =============================================================================
# 19. CLOSURE
# =============================================================================

banner(
    "STAGE26-4B3 CPU1 EXTRACTION TIMING COMPLETE"
)


print(
    "STATUS:"
)

print(
    " ",
    receipt_status
)


print(
    "\nFROZEN CONDITION:"
)

print(
    "  source      : Monday-WorkingHours.pcap"
)

print(
    "  packet range: 1 .. 1,000,000"
)

print(
    "  CPU affinity: [0]"
)

print(
    "  repetitions :",
    REPETITIONS
)

print(
    "  successful  :",
    len(
        pass_records
    )
)


if pass_records:

    print(
        "\nPRIMARY SUMMARY:"
    )


    for metric in [
        "packets_per_second",
        "MiB_per_second",
        "flows_per_second",
        "elapsed_seconds",
    ]:

        values = summary_metrics[
            metric
        ]


        print(
            f"  {metric:24s} "
            f"median={values['median']:,.6f}  "
            f"min={values['minimum']:,.6f}  "
            f"max={values['maximum']:,.6f}"
        )


print(
    "\nPREFIX COUNT FINGERPRINT:"
)

print(
    " ",
    prefix_count_fingerprint
)


print(
    "\nSUMMARY JSON:"
)

print(
    " ",
    SUMMARY_JSON
)

print(
    " SHA256:",
    summary_sha
)


print(
    "\nSUMMARY CSV:"
)

print(
    " ",
    SUMMARY_CSV
)

print(
    " SHA256:",
    csv_sha
)


print(
    "\nRECEIPT:"
)

print(
    " ",
    RECEIPT_JSON
)

print(
    " SHA256:",
    receipt_sha
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  FULL-MONDAY CORRECTNESS GATE       : PASS"
)

print(
    "  EXTRACTION PERFORMANCE MEASURED    : YES"
)

print(
    "  RAW PREFIX                         : 1,000,000 packets"
)

print(
    "  CPU                                : CPU1"
)

print(
    "  RELEASE CORPUS REGENERATED         : NO"
)

print(
    "  NEW CORPUS CREATED                 : NO"
)

print(
    "  REPRESENTATION TIMING              : NO"
)

print(
    "  MODELS LOADED                      : NO"
)

print(
    "  COMPLETE E2E CLAIM                 : NO"
)

print(
    "  GPU                                : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Anchor Stage26-4B3 raw timing outputs to GitHub"
)

print(
    "  before proceeding to any additional Stage26 profiling."
)


STAGE26-4B3 :: DURABLE SCIENTIFIC GATE
Expected HEAD : e558da134162652c1a955f60f15c175f083923c8
Local HEAD    : e558da134162652c1a955f60f15c175f083923c8
origin/main   : e558da134162652c1a955f60f15c175f083923c8
Repo clean    : True

STAGE26-4B3 :: FROZEN PROVENANCE HASH GATE
worker                       PASS ba34c5ab1a8e0bdb22acf5d6be3430cc2b31fdec04dc333dcf3ef7344abe44e6
protocol                     PASS 12f7f8f0332d92694a827e3473dc1cef66939cf1cf1c49c17ad1699e60d929b0
protocol manifest            PASS 7fe04d2628e1bfe7da5551192f45821b5406b3b3087976c4334bcd74224fc2d3
validation result            PASS a012ba81b4cbb339371898e6adc4cd43a1c6364086d1b343f0fb7cd73c3bb46f
validation receipt           PASS 0634cd71d3b1d2111a7b44a3519711660fb6ec3ad8f143804ba1cd3b9e6802d2
validation manifest          PASS 87cc9f272e451a6ea032a1aa12f147bea311f80eadce0cfb51ef8f734efccd32

STAGE26-4B3 :: TIMING PERMISSION GATE
full_validation_exact                 : PASS
all_frozen_counts_exact               : PASS
z

In [18]:
# =============================================================================
# STAGE26-4B3-GIT
# DURABLY ANCHOR CPU1 RAW EXTRACTION TIMING
#
# CURRENT SCIENTIFIC PARENT:
#   e558da134162652c1a955f60f15c175f083923c8
#
# COMPLETED MEASUREMENT:
#   5 / 5 frozen CPU1 repetitions PASS
#   raw packet prefix 1 .. 1,000,000
#   deterministic count fingerprint across all repetitions
#
# THIS CELL ONLY:
#   - verifies the completed Stage26-4B3 outputs;
#   - copies ALL raw configs/results + summaries into the repo;
#   - creates a durable manifest;
#   - commits;
#   - pushes;
#   - fetches;
#   - remotely byte-verifies every committed measurement artifact.
#
# ABSOLUTE RULES:
#   - NO extraction remeasurement.
#   - NO PCAP iteration.
#   - NO PCAP rehash.
#   - NO corpus creation/regeneration.
#   - NO representation timing.
#   - NO model loading.
#   - NO holdout.
#   - NO GPU.
#   - NO scientific interpretation beyond integrity checks.
# =============================================================================

from __future__ import annotations

import os
import json
import csv
import hashlib
import shutil
import stat
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. FROZEN IDENTITIES / PATHS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

SOURCE_RUN_ROOT = (
    STAGE26_ROOT
    / "extraction"
    / "stage26_4b3_cpu1_extraction_timing"
)

EXPECTED_PARENT = (
    "e558da134162652c1a955f60f15c175f083923c8"
)

EXPECTED_PROTOCOL_COMMIT = (
    "b41a0559598a70e6c69aac05c76e4c4edb73b34e"
)

COMMIT_SUBJECT = (
    "stage26: anchor CPU1 raw extraction timing"
)


# -----------------------------------------------------------------------------
# Frozen Stage26-4B3 top-level output identities
# -----------------------------------------------------------------------------

EXPECTED_SUMMARY_JSON_SHA256 = (
    "1914504fad850879d1ed11afed432436c18454f2bb8650f881244c865f18ba16"
)

EXPECTED_SUMMARY_CSV_SHA256 = (
    "4df7e43c658cd6447bb505fae3b2e68b5868be18701849f93f4a6e59af951285"
)

EXPECTED_RECEIPT_SHA256 = (
    "38e4faa798cdf038d401191190103e81e02ed5c20bb10417e90e28343e56649e"
)

EXPECTED_PREFIX_COUNT_FINGERPRINT = (
    "8672e244f8f87cc79a3667c002dd28c8d20c221b02825a9ed5e77d643a8cd171"
)


SOURCE_SUMMARY_JSON = (
    SOURCE_RUN_ROOT
    / "stage26_4b3_cpu1_extraction_summary.json"
)

SOURCE_SUMMARY_CSV = (
    SOURCE_RUN_ROOT
    / "stage26_4b3_cpu1_extraction_repetitions.csv"
)

SOURCE_RECEIPT = (
    SOURCE_RUN_ROOT
    / "stage26_4b3_cpu1_extraction_receipt.json"
)


# -----------------------------------------------------------------------------
# Durable repository checkpoint
# -----------------------------------------------------------------------------

CHECKPOINT_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_4b3_cpu1_extraction_timing"
)

CHECKPOINT_DIR = (
    REPO
    / CHECKPOINT_REL
)

MANIFEST_PATH = (
    CHECKPOINT_DIR
    / "stage26_4b3_cpu1_extraction_manifest.json"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 122
    )

    print(text)

    print(
        "=" * 122
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        check=True,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        check=True,
        text=False,
    )

    return bytes(
        p.stdout
    )


# =============================================================================
# 2. SCIENTIFIC PARENT GATE
# =============================================================================

banner(
    "STAGE26-4B3-GIT :: SCIENTIFIC PARENT GATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected local HEAD before Stage26-4B3 anchor."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Stage26-4B3 anchor."
    )


if status:

    raise RuntimeError(
        "Repository must be clean before Stage26-4B3 anchor."
    )


# =============================================================================
# 3. SOURCE MEASUREMENT DIRECTORY GATE
# =============================================================================

banner(
    "STAGE26-4B3-GIT :: SOURCE OUTPUT INVENTORY"
)


if not SOURCE_RUN_ROOT.exists():

    raise FileNotFoundError(
        SOURCE_RUN_ROOT
    )


source_files = sorted(
    p
    for p in SOURCE_RUN_ROOT.iterdir()
    if p.is_file()
)


print(
    "Source files:",
    len(
        source_files
    )
)


for p in source_files:

    print(
        f"  {p.name:56s} "
        f"{p.stat().st_size:10,d} B"
    )


# Frozen run should contain:
#
#   5 config JSON
#   5 result JSON
#   1 summary JSON
#   1 summary CSV
#   1 receipt JSON
#
# = 13 total files.

expected_names = {
    *(f"rep_{i:02d}_config.json" for i in range(1, 6)),
    *(f"rep_{i:02d}_result.json" for i in range(1, 6)),
    "stage26_4b3_cpu1_extraction_summary.json",
    "stage26_4b3_cpu1_extraction_repetitions.csv",
    "stage26_4b3_cpu1_extraction_receipt.json",
}


actual_names = {
    p.name
    for p in source_files
}


if actual_names != expected_names:

    print(
        "\nExpected:"
    )

    for name in sorted(
        expected_names
    ):

        print(
            " ",
            name
        )


    print(
        "\nActual:"
    )

    for name in sorted(
        actual_names
    ):

        print(
            " ",
            name
        )


    raise RuntimeError(
        "Unexpected Stage26-4B3 source output set."
    )


print(
    "\n[PASS] Exact 13-file Stage26-4B3 output set."
)


# =============================================================================
# 4. TOP-LEVEL HASH GATE
# =============================================================================

banner(
    "STAGE26-4B3-GIT :: TOP-LEVEL HASH GATE"
)


top_hashes = [
    (
        "summary JSON",
        SOURCE_SUMMARY_JSON,
        EXPECTED_SUMMARY_JSON_SHA256,
    ),
    (
        "summary CSV",
        SOURCE_SUMMARY_CSV,
        EXPECTED_SUMMARY_CSV_SHA256,
    ),
    (
        "receipt",
        SOURCE_RECEIPT,
        EXPECTED_RECEIPT_SHA256,
    ),
]


for label, path, expected in top_hashes:

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )

    print(
        f"{label:18s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )

    if not passed:

        raise RuntimeError(
            f"Stage26-4B3 {label} changed."
        )


# =============================================================================
# 5. SCIENTIFIC SUMMARY GATE
# =============================================================================

banner(
    "STAGE26-4B3-GIT :: SCIENTIFIC SUMMARY GATE"
)


summary = json.loads(
    SOURCE_SUMMARY_JSON.read_text(
        encoding="utf-8"
    )
)

receipt = json.loads(
    SOURCE_RECEIPT.read_text(
        encoding="utf-8"
    )
)


checks = {
    "checkpoint":
        (
            summary[
                "checkpoint"
            ]
            ==
            "STAGE26-4B3"
        ),

    "scientific_parent":
        (
            summary[
                "scientific_parent"
            ]
            ==
            EXPECTED_PARENT
        ),

    "protocol_commit":
        (
            summary[
                "protocol_commit"
            ]
            ==
            EXPECTED_PROTOCOL_COMMIT
        ),

    "measurement_complete":
        (
            summary[
                "measurement_complete"
            ]
            is True
        ),

    "successful_repetitions_5":
        (
            summary[
                "successful_repetitions"
            ]
            ==
            5
        ),

    "PASS_count_5":
        (
            summary[
                "status_counts"
            ].get(
                "PASS",
                0,
            )
            ==
            5
        ),

    "prefix_count_consistency":
        (
            summary[
                "prefix_count_consistency"
            ]
            is True
        ),

    "prefix_fingerprint":
        (
            summary[
                "prefix_count_summary_sha256"
            ]
            ==
            EXPECTED_PREFIX_COUNT_FINGERPRINT
        ),

    "packet_count_1M":
        (
            summary[
                "condition"
            ][
                "packet_count"
            ]
            ==
            1_000_000
        ),

    "packet_start_1":
        (
            summary[
                "condition"
            ][
                "packet_start"
            ]
            ==
            1
        ),

    "packet_end_1M":
        (
            summary[
                "condition"
            ][
                "packet_end"
            ]
            ==
            1_000_000
        ),

    "affinity_CPU0":
        (
            summary[
                "condition"
            ][
                "affinity"
            ]
            ==
            [0]
        ),

    "thread_count_1":
        (
            summary[
                "condition"
            ][
                "thread_count"
            ]
            ==
            1
        ),

    "fresh_process":
        (
            summary[
                "condition"
            ][
                "fresh_process_per_repetition"
            ]
            is True
        ),

    "disk_IO_included":
        (
            summary[
                "condition"
            ][
                "disk_IO_included"
            ]
            is True
        ),

    "serialization_excluded":
        (
            summary[
                "condition"
            ][
                "serialization_included"
            ]
            is False
        ),

    "Python_startup_excluded":
        (
            summary[
                "condition"
            ][
                "Python_process_startup_included"
            ]
            is False
        ),

    "JVM_startup_excluded":
        (
            summary[
                "condition"
            ][
                "JVM_startup_included"
            ]
            is False
        ),

    "page_cache_not_manipulated":
        (
            summary[
                "condition"
            ][
                "page_cache_manipulated"
            ]
            is False
        ),

    "GPU_false":
        (
            summary[
                "condition"
            ][
                "gpu"
            ]
            is False
        ),

    "corpus_not_created":
        (
            summary[
                "scientific_boundaries"
            ][
                "corpus_created"
            ]
            is False
        ),

    "release_not_regenerated":
        (
            summary[
                "scientific_boundaries"
            ][
                "release_corpus_regenerated"
            ]
            is False
        ),

    "representation_timing_false":
        (
            summary[
                "scientific_boundaries"
            ][
                "representation_timing"
            ]
            is False
        ),

    "E2E_claim_false":
        (
            summary[
                "scientific_boundaries"
            ][
                "complete_end_to_end_claim"
            ]
            is False
        ),

    "receipt_status":
        (
            receipt[
                "status"
            ]
            ==
            "PASS_FIVE_FROZEN_REPETITIONS"
        ),

    "receipt_success_5":
        (
            receipt[
                "successful_repetitions"
            ]
            ==
            5
        ),

    "receipt_fingerprint":
        (
            receipt[
                "prefix_count_summary_sha256"
            ]
            ==
            EXPECTED_PREFIX_COUNT_FINGERPRINT
        ),

    "receipt_GPU_false":
        (
            receipt[
                "gpu_used"
            ]
            is False
        ),
}


for name, passed in checks.items():

    print(
        f"{name:42s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    checks.values()
):

    failed = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Scientific summary gate failed:\n"
        +
        "\n".join(
            failed
        )
    )


# =============================================================================
# 6. VERIFY ALL FIVE RAW REPETITIONS
# =============================================================================

banner(
    "STAGE26-4B3-GIT :: RAW REPETITION AUDIT"
)


repetition_rows = []


for i in range(
    1,
    6,
):

    config_path = (
        SOURCE_RUN_ROOT
        / f"rep_{i:02d}_config.json"
    )

    result_path = (
        SOURCE_RUN_ROOT
        / f"rep_{i:02d}_result.json"
    )


    config = json.loads(
        config_path.read_text(
            encoding="utf-8"
        )
    )

    result = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


    config_sha = sha256_file(
        config_path
    )

    result_sha = sha256_file(
        result_path
    )


    rep_checks = {
        "rep_index":
            (
                config[
                    "repetition"
                ]
                ==
                i
        ),

        "condition":
            (
                config[
                    "condition"
                ]
                ==
                "CPU1_RAW_PACKET_PREFIX_1000000"
        ),

        "mode":
            (
                config[
                    "mode"
                ]
                ==
                "BENCHMARK_TIMED"
        ),

        "affinity":
            (
                config[
                    "affinity"
                ]
                ==
                [0]
        ),

        "packet_limit":
            (
                config[
                    "packet_limit"
                ]
                ==
                1_000_000
        ),

        "result_status":
            (
                result[
                    "status"
                ]
                ==
                "PASS"
        ),

        "result_mode":
            (
                result[
                    "mode"
                ]
                ==
                "BENCHMARK_TIMED"
        ),

        "timing_performed":
            (
                result[
                    "timing_performed"
                ]
                is True
        ),

        "raw_packets":
            (
                result[
                    "counts"
                ][
                    "raw_packet_count"
                ]
                ==
                1_000_000
        ),

        "count_fingerprint":
            (
                result[
                    "count_summary_sha256"
                ]
                ==
                EXPECTED_PREFIX_COUNT_FINGERPRINT
        ),

        "corpus_false":
            (
                result[
                    "corpus_created_or_modified"
                ]
                is False
        ),

        "labels_false":
            (
                result[
                    "labels_accessed"
                ]
                is False
        ),

        "gpu_false":
            (
                result[
                    "gpu_used"
                ]
                is False
        ),
    }


    if not all(
        rep_checks.values()
    ):

        failed = [
            name
            for name, passed in rep_checks.items()
            if not passed
        ]

        raise RuntimeError(
            f"Repetition {i} gate failed: "
            +
            ", ".join(
                failed
            )
        )


    # Count dictionary fingerprint independently.
    recomputed_count_sha = hashlib.sha256(
        json.dumps(
            result[
                "counts"
            ],
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
        ).encode(
            "utf-8"
        )
    ).hexdigest()


    if (
        recomputed_count_sha
        !=
        EXPECTED_PREFIX_COUNT_FINGERPRINT
    ):

        raise RuntimeError(
            f"Rep {i}: recomputed count fingerprint mismatch."
        )


    repetition_rows.append(
        {
            "repetition":
                i,

            "config_sha256":
                config_sha,

            "result_sha256":
                result_sha,

            "elapsed_seconds":
                float(
                    result[
                        "elapsed_seconds"
                    ]
                ),

            "packets_per_second":
                float(
                    result[
                        "packets_per_second"
                    ]
                ),

            "MiB_per_second":
                float(
                    result[
                        "MiB_per_second"
                    ]
                ),

            "flows_per_second":
                float(
                    result[
                        "flows_per_second"
                    ]
                ),

            "exportable_flow_count":
                int(
                    result[
                        "counts"
                    ][
                        "exportable_flow_count"
                    ]
                ),

            "count_summary_sha256":
                result[
                    "count_summary_sha256"
                ],
        }
    )


    print(
        f"REP {i}: PASS | "
        f"{result['elapsed_seconds']:.6f} s | "
        f"{result['packets_per_second']:,.2f} pkt/s | "
        f"{result['MiB_per_second']:,.2f} MiB/s | "
        f"{result['flows_per_second']:,.2f} flow/s | "
        f"flows={result['counts']['exportable_flow_count']:,}"
    )


# =============================================================================
# 7. CHECK SUMMARY RAW VALUES AGAINST RESULT FILES
# =============================================================================

banner(
    "STAGE26-4B3-GIT :: SUMMARY / RAW CONSISTENCY"
)


summary_reps = {
    int(
        row[
            "repetition"
        ]
    ):
        row
    for row in summary[
        "repetitions"
    ]
}


if set(
    summary_reps
) != set(
    range(
        1,
        6,
    )
):

    raise RuntimeError(
        "Summary does not contain exactly repetitions 1..5."
    )


for raw in repetition_rows:

    i = raw[
        "repetition"
    ]

    sr = summary_reps[
        i
    ]


    for field in [
        "elapsed_seconds",
        "packets_per_second",
        "MiB_per_second",
        "flows_per_second",
    ]:

        if float(
            sr[
                field
            ]
        ) != float(
            raw[
                field
            ]
        ):

            raise RuntimeError(
                f"Rep {i}: summary/raw mismatch in {field}."
            )


    if (
        sr[
            "count_summary_sha256"
        ]
        !=
        EXPECTED_PREFIX_COUNT_FINGERPRINT
    ):

        raise RuntimeError(
            f"Rep {i}: summary fingerprint mismatch."
        )


print(
    "Summary repetition rows exactly reproduce raw result values: PASS"
)


# =============================================================================
# 8. CSV CONSISTENCY GATE
# =============================================================================

banner(
    "STAGE26-4B3-GIT :: CSV CONSISTENCY"
)


with SOURCE_SUMMARY_CSV.open(
    "r",
    encoding="utf-8",
    newline="",
) as f:

    csv_rows = list(
        csv.DictReader(
            f
        )
    )


print(
    "CSV repetition rows:",
    len(
        csv_rows
    )
)


if len(
    csv_rows
) != 5:

    raise RuntimeError(
        "Expected exactly five CSV repetition rows."
    )


for row in csv_rows:

    if row[
        "status"
    ] != "PASS":

        raise RuntimeError(
            "Non-PASS row found in frozen CSV."
        )


    if row[
        "count_summary_sha256"
    ] != EXPECTED_PREFIX_COUNT_FINGERPRINT:

        raise RuntimeError(
            "CSV count fingerprint mismatch."
        )


print(
    "[PASS] CSV contains 5/5 PASS repetitions with frozen fingerprint."
)


# =============================================================================
# 9. DISPLAY FROZEN PRIMARY SUMMARY — NO NEW INFERENCE
# =============================================================================

banner(
    "STAGE26-4B3-GIT :: FROZEN PRIMARY SUMMARY"
)


for metric in [
    "elapsed_seconds",
    "packets_per_second",
    "MiB_per_second",
    "flows_per_second",
]:

    values = summary[
        "metrics"
    ][
        metric
    ]


    print(
        f"{metric:24s} "
        f"median={values['median']:,.6f}  "
        f"min={values['minimum']:,.6f}  "
        f"max={values['maximum']:,.6f}"
    )


# =============================================================================
# 10. BUILD DURABLE CHECKPOINT — COPY ONLY, NO REMEASUREMENT
# =============================================================================

banner(
    "STAGE26-4B3-GIT :: BUILD DURABLE CHECKPOINT"
)


if CHECKPOINT_DIR.exists():

    raise RuntimeError(
        "Durable Stage26-4B3 checkpoint already exists. "
        "Do not overwrite."
    )


CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


for source in source_files:

    destination = (
        CHECKPOINT_DIR
        / source.name
    )


    shutil.copy2(
        source,
        destination,
    )


print(
    "Copied frozen measurement files:",
    len(
        source_files
    )
)


# =============================================================================
# 11. CREATE DURABLE MANIFEST
# =============================================================================

manifest_rows = []


for path in sorted(
    CHECKPOINT_DIR.iterdir()
):

    if (
        path.is_file()
        and
        path != MANIFEST_PATH
    ):

        manifest_rows.append(
            {
                "repo_relative_path":
                    str(
                        path.relative_to(
                            REPO
                        )
                    ),

                "size_bytes":
                    int(
                        path.stat().st_size
                    ),

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )


if len(
    manifest_rows
) != 13:

    raise RuntimeError(
        "Expected exactly 13 frozen measurement files before manifest."
    )


manifest = {
    "schema":
        "stage26_4b3_cpu1_extraction_manifest_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4B3",

    "status":
        "PASS_FIVE_FROZEN_REPETITIONS_READY_FOR_GIT_ANCHOR",

    "parent_commit":
        EXPECTED_PARENT,

    "protocol_commit":
        EXPECTED_PROTOCOL_COMMIT,

    "measurement": {
        "source":
            "Monday-WorkingHours.pcap",

        "packet_sampling_rule":
            "DETERMINISTIC_CONTIGUOUS_RAW_PACKET_PREFIX",

        "packet_start":
            1,

        "packet_end":
            1_000_000,

        "packet_count":
            1_000_000,

        "cpu_mode":
            "CPU_1_PHYSICAL_CORE",

        "affinity":
            [0],

        "thread_count":
            1,

        "requested_repetitions":
            5,

        "successful_repetitions":
            5,

        "fresh_process_per_repetition":
            True,

        "disk_IO_included":
            True,

        "serialization_included":
            False,

        "Python_process_startup_included":
            False,

        "JVM_startup_included":
            False,

        "page_cache_manipulated":
            False,
    },

    "determinism": {
        "all_successful_count_dicts_identical":
            True,

        "prefix_count_summary_sha256":
            EXPECTED_PREFIX_COUNT_FINGERPRINT,
    },

    "primary_summary": {
        metric:
            summary[
                "metrics"
            ][
                metric
            ]
        for metric in [
            "elapsed_seconds",
            "packets_per_second",
            "MiB_per_second",
            "flows_per_second",
        ]
    },

    "top_level_hashes": {
        "summary_json_sha256":
            EXPECTED_SUMMARY_JSON_SHA256,

        "summary_csv_sha256":
            EXPECTED_SUMMARY_CSV_SHA256,

        "receipt_sha256":
            EXPECTED_RECEIPT_SHA256,
    },

    "scientific_boundaries": {
        "raw_extraction_performance_measured":
            True,

        "corpus_created":
            False,

        "release_corpus_regenerated":
            False,

        "representation_timing":
            False,

        "model_inference_performed":
            False,

        "complete_end_to_end_claim":
            False,

        "gpu_used":
            False,
    },

    "raw_repetitions":
        repetition_rows,

    "file_count_excluding_manifest":
        len(
            manifest_rows
        ),

    "files":
        manifest_rows,
}


atomic_json(
    MANIFEST_PATH,
    manifest,
)


manifest_sha = sha256_file(
    MANIFEST_PATH
)


print(
    "Manifest:"
)

print(
    " ",
    MANIFEST_PATH
)

print(
    "SHA256:"
)

print(
    " ",
    manifest_sha
)


# =============================================================================
# 12. LOCAL MANIFEST SELF-AUDIT
# =============================================================================

banner(
    "STAGE26-4B3-GIT :: LOCAL MANIFEST SELF-AUDIT"
)


for row in manifest[
    "files"
]:

    path = (
        REPO
        / row[
            "repo_relative_path"
        ]
    )


    actual_size = int(
        path.stat().st_size
    )

    actual_sha = sha256_file(
        path
    )


    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Local durable checkpoint manifest mismatch."
        )


# =============================================================================
# 13. REPOSITORY CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-4B3-GIT :: REPOSITORY CHANGE AUDIT"
)


status_after_build = git(
    "status",
    "--porcelain",
)


print(
    status_after_build
)


if not status_after_build:

    raise RuntimeError(
        "Expected uncommitted Stage26-4B3 checkpoint."
    )


unexpected = []


for line in status_after_build.splitlines():

    rel = line[
        3:
    ]


    if not rel.startswith(
        str(
            CHECKPOINT_REL
        )
        +
        "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected Git modifications outside Stage26-4B3:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 14. COMMIT
# =============================================================================

banner(
    "STAGE26-4B3-GIT :: COMMIT"
)


git(
    "add",
    str(
        CHECKPOINT_REL
    ),
)


staged = git(
    "diff",
    "--cached",
    "--name-status",
)


print(
    staged
)


if not staged:

    raise RuntimeError(
        "Nothing staged for Stage26-4B3."
    )


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-4B3 commit parent mismatch."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Stage26-4B3 commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository is not clean after Stage26-4B3 commit."
    )


# =============================================================================
# 15. PUSH USING KAGGLE GITHUB_TOKEN
# =============================================================================

banner(
    "STAGE26-4B3-GIT :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle secret GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    push = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
        check=True,
        text=True,
    )


    print(
        push.stdout.strip()
    )


github_token = None


# =============================================================================
# 16. FETCH + REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-4B3-GIT :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed unexpectedly."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "origin/main does not equal Stage26-4B3 commit."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote Stage26-4B3 parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote Stage26-4B3 subject mismatch."
    )


# =============================================================================
# 17. REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-4B3-GIT :: REMOTE BYTE VERIFICATION"
)


manifest_rel = str(
    MANIFEST_PATH.relative_to(
        REPO
    )
)


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    manifest_rel,
)

remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "Remote manifest SHA256:"
)

print(
    " ",
    remote_manifest_sha
)


if remote_manifest_sha != manifest_sha:

    raise RuntimeError(
        "Remote Stage26-4B3 manifest SHA mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


remote_file_count = 0


for row in remote_manifest[
    "files"
]:

    rel = row[
        "repo_relative_path"
    ]

    data = git_blob_bytes(
        "origin/main",
        rel,
    )

    actual_size = len(
        data
    )

    actual_sha = sha256_bytes(
        data
    )


    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{rel}"
    )


    if not passed:

        raise RuntimeError(
            f"Remote byte verification failed: {rel}"
        )


    remote_file_count += 1


print(
    "\nManifest-listed files remotely verified:",
    remote_file_count
)


if remote_file_count != 13:

    raise RuntimeError(
        "Expected 13 remotely verified measurement files."
    )


# =============================================================================
# 18. REMOTE SCIENTIFIC CONTENT AUDIT
# =============================================================================

banner(
    "STAGE26-4B3-GIT :: REMOTE SCIENTIFIC CONTENT AUDIT"
)


remote_summary_rel = str(
    CHECKPOINT_REL
    /
    SOURCE_SUMMARY_JSON.name
)

remote_receipt_rel = str(
    CHECKPOINT_REL
    /
    SOURCE_RECEIPT.name
)


remote_summary = json.loads(
    git_blob_bytes(
        "origin/main",
        remote_summary_rel,
    ).decode(
        "utf-8"
    )
)


remote_receipt = json.loads(
    git_blob_bytes(
        "origin/main",
        remote_receipt_rel,
    ).decode(
        "utf-8"
    )
)


remote_checks = {
    "measurement_complete":
        (
            remote_summary[
                "measurement_complete"
            ]
            is True
        ),

    "five_successful_repetitions":
        (
            remote_summary[
                "successful_repetitions"
            ]
            ==
            5
        ),

    "five_PASS":
        (
            remote_summary[
                "status_counts"
            ].get(
                "PASS",
                0,
            )
            ==
            5
        ),

    "deterministic_counts":
        (
            remote_summary[
                "prefix_count_consistency"
            ]
            is True
        ),

    "prefix_fingerprint":
        (
            remote_summary[
                "prefix_count_summary_sha256"
            ]
            ==
            EXPECTED_PREFIX_COUNT_FINGERPRINT
        ),

    "packet_prefix_1M":
        (
            remote_summary[
                "condition"
            ][
                "packet_start"
            ]
            ==
            1
            and
            remote_summary[
                "condition"
            ][
                "packet_end"
            ]
            ==
            1_000_000
        ),

    "CPU1":
        (
            remote_summary[
                "condition"
            ][
                "affinity"
            ]
            ==
            [0]
            and
            remote_summary[
                "condition"
            ][
                "thread_count"
            ]
            ==
            1
        ),

    "disk_IO_included":
        (
            remote_summary[
                "condition"
            ][
                "disk_IO_included"
            ]
            is True
        ),

    "corpus_not_created":
        (
            remote_summary[
                "scientific_boundaries"
            ][
                "corpus_created"
            ]
            is False
        ),

    "release_not_regenerated":
        (
            remote_summary[
                "scientific_boundaries"
            ][
                "release_corpus_regenerated"
            ]
            is False
        ),

    "representation_timing_false":
        (
            remote_summary[
                "scientific_boundaries"
            ][
                "representation_timing"
            ]
            is False
        ),

    "complete_E2E_claim_false":
        (
            remote_summary[
                "scientific_boundaries"
            ][
                "complete_end_to_end_claim"
            ]
            is False
        ),

    "GPU_false":
        (
            remote_summary[
                "scientific_boundaries"
            ][
                "gpu_used"
            ]
            is False
        ),

    "receipt_status":
        (
            remote_receipt[
                "status"
            ]
            ==
            "PASS_FIVE_FROZEN_REPETITIONS"
        ),
}


for name, passed in remote_checks.items():

    print(
        f"{name:40s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    remote_checks.values()
):

    raise RuntimeError(
        "Remote Stage26-4B3 scientific content audit failed."
    )


# =============================================================================
# 19. FINAL REPOSITORY AUDIT
# =============================================================================

banner(
    "STAGE26-4B3-GIT :: FINAL AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != final_remote:

    raise RuntimeError(
        "Local/remote divergence after Stage26-4B3 anchor."
    )


if final_status:

    raise RuntimeError(
        "Repository not clean after Stage26-4B3 anchor."
    )


# =============================================================================
# 20. CLOSURE
# =============================================================================

banner(
    "STAGE26-4B3-GIT COMPLETE"
)


print(
    "COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nSUBJECT:"
)

print(
    " ",
    COMMIT_SUBJECT
)


print(
    "\nPARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nRAW MEASUREMENT:"
)

print(
    "  repetitions : 5 / 5 PASS"
)

print(
    "  packet range: 1 .. 1,000,000"
)

print(
    "  CPU         : affinity [0], 1 thread"
)

print(
    "  fingerprint :",
    EXPECTED_PREFIX_COUNT_FINGERPRINT
)


print(
    "\nPRIMARY FROZEN SUMMARY:"
)


for metric in [
    "packets_per_second",
    "MiB_per_second",
    "flows_per_second",
    "elapsed_seconds",
]:

    values = summary[
        "metrics"
    ][
        metric
    ]


    print(
        f"  {metric:22s} "
        f"median={values['median']:,.6f} "
        f"min={values['minimum']:,.6f} "
        f"max={values['maximum']:,.6f}"
    )


print(
    "\nDURABLE HASHES:"
)

print(
    "  source summary JSON:",
    EXPECTED_SUMMARY_JSON_SHA256
)

print(
    "  source summary CSV :",
    EXPECTED_SUMMARY_CSV_SHA256
)

print(
    "  source receipt     :",
    EXPECTED_RECEIPT_SHA256
)

print(
    "  package manifest   :",
    manifest_sha
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  local == origin/main       : PASS"
)

print(
    "  commit parent              : PASS"
)

print(
    "  all 13 raw measurement files: PASS"
)

print(
    "  manifest                   : PASS"
)

print(
    "  scientific content         : PASS"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  RAW EXTRACTION CPU1 TIMING ANCHORED : YES"
)

print(
    "  5/5 REPETITIONS                    : PASS"
)

print(
    "  DETERMINISTIC PREFIX COUNTS         : YES"
)

print(
    "  CORPUS CREATED                      : NO"
)

print(
    "  RELEASE CORPUS REGENERATED          : NO"
)

print(
    "  REPRESENTATION TIMING               : NO"
)

print(
    "  MODEL INFERENCE                     : NO"
)

print(
    "  COMPLETE END-TO-END CLAIM           : NO"
)

print(
    "  GPU                                 : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Freeze the separate Release-corpus -> representation"
)

print(
    "  profiling boundary before measuring Stage26-4C."
)


STAGE26-4B3-GIT :: SCIENTIFIC PARENT GATE
Expected parent: e558da134162652c1a955f60f15c175f083923c8
Local HEAD     : e558da134162652c1a955f60f15c175f083923c8
origin/main    : e558da134162652c1a955f60f15c175f083923c8
Repo clean     : True

STAGE26-4B3-GIT :: SOURCE OUTPUT INVENTORY
Source files: 13
  rep_01_config.json                                            1,126 B
  rep_01_result.json                                            2,104 B
  rep_02_config.json                                            1,125 B
  rep_02_result.json                                            2,102 B
  rep_03_config.json                                            1,126 B
  rep_03_result.json                                            2,104 B
  rep_04_config.json                                            1,126 B
  rep_04_result.json                                            2,104 B
  rep_05_config.json                                            1,125 B
  rep_05_result.json                                

In [19]:
# =============================================================================
# STAGE26-4C0
# RELEASE-CORPUS -> REPRESENTATION IMPLEMENTATION INVENTORY
#
# DURABLE SCIENTIFIC PARENT:
#   55aba2e1cc08385659479c6275234dc23b11d231
#
# PURPOSE:
#   Determine the EXACT existing Stage20 representation implementation and
#   scientific boundary BEFORE freezing any Stage26-4C timing protocol.
#
# THIS CELL:
#   - anchors Git to the completed raw-extraction commit;
#   - verifies Stage26-4B3 remains durable;
#   - verifies the existing Monday GitHub Release corpus TAR;
#   - inventories exact Stage20 packet-image encoder source;
#   - searches repository history/artifacts for representation geometry,
#     masking, truncation, /255 scaling, reshape/materialization semantics;
#   - prints relevant source excerpts;
#   - inventories functions/classes in the encoder using Python AST;
#   - inspects .npy headers/geometry from the existing Release TAR;
#   - writes ONE local inventory receipt outside Git.
#
# ABSOLUTE RULES:
#   - NO representation timing.
#   - NO PCAP access.
#   - NO corpus recreation.
#   - NO model loading.
#   - NO inference.
#   - NO Thursday/Friday.
#   - NO GPU.
#   - NO Git modification.
#
# IMPORTANT:
#   This cell does not yet freeze Stage26-4C.
#   We first establish the exact implementation boundary from existing code.
# =============================================================================

from __future__ import annotations

import ast
import io
import os
import re
import json
import hashlib
import tarfile
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np


# =============================================================================
# 0. PATHS / FROZEN IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

EXTRACTION_ROOT = (
    STAGE26_ROOT
    / "extraction"
)

REP_ROOT = (
    STAGE26_ROOT
    / "representation"
)

REP_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


EXPECTED_HEAD = (
    "55aba2e1cc08385659479c6275234dc23b11d231"
)


# -----------------------------------------------------------------------------
# Completed raw extraction durable checkpoint
# -----------------------------------------------------------------------------

RAW_TIMING_ROOT = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4b3_cpu1_extraction_timing"
)

RAW_TIMING_MANIFEST = (
    RAW_TIMING_ROOT
    / "stage26_4b3_cpu1_extraction_manifest.json"
)

EXPECTED_RAW_TIMING_MANIFEST_SHA256 = (
    "39900c940ca25f3a541efe0dcb5f08d51408581f1f6f4b93980b093327276bc1"
)


# -----------------------------------------------------------------------------
# Existing authoritative Monday Release corpus
# -----------------------------------------------------------------------------

MONDAY_TAR = (
    STAGE26_ROOT
    / "release_corpora"
    / "Monday"
    / "stage20-Monday-compact-corpus-v1.tar"
)

EXPECTED_TAR_SIZE = (
    595_261_440
)

EXPECTED_TAR_SHA256 = (
    "4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20"
)

EXPECTED_MEMBERS = {
    "Monday/encoded_bytes.bin":
        "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",

    "Monday/flow_offsets.npy":
        "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",

    "Monday/labels.npy":
        "48792b8d6a127b35342cb0789baa6c54396f1100a60ce7225daf08d1c3530424",

    "Monday/packet_lengths.npy":
        "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
}


# -----------------------------------------------------------------------------
# Exact known Stage20 encoder path
# -----------------------------------------------------------------------------

ENCODER = (
    REPO
    / "scripts"
    / "stage20_packet_image_encoder.py"
)


OUTPUT_RECEIPT = (
    REP_ROOT
    / "stage26_4c0_representation_implementation_inventory.json"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 122
    )

    print(text)

    print(
        "=" * 122
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            p.stdout
        )

    return p


def git(*args):

    return run(
        [
            "git",
            *args,
        ]
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def human_bytes(n):

    value = float(
        n
    )

    for unit in [
        "B",
        "KiB",
        "MiB",
        "GiB",
        "TiB",
    ]:

        if (
            value < 1024
            or
            unit == "TiB"
        ):

            return (
                f"{value:.3f} {unit}"
            )

        value /= 1024


def source_excerpt(
    text,
    start_line,
    end_line,
):

    lines = text.splitlines()

    start_line = max(
        1,
        start_line,
    )

    end_line = min(
        len(
            lines
        ),
        end_line,
    )


    return "\n".join(
        f"{i:5d}: {lines[i - 1]}"
        for i in range(
            start_line,
            end_line + 1,
        )
    )


# =============================================================================
# 2. SCIENTIFIC GIT ANCHOR
# =============================================================================

banner(
    "STAGE26-4C0 :: SCIENTIFIC ANCHOR"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD :",
    EXPECTED_HEAD
)

print(
    "Local HEAD    :",
    head
)

print(
    "origin/main   :",
    remote
)

print(
    "Repo clean    :",
    status == ""
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected Stage26 parent."
    )


if remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed."
    )


if status:

    raise RuntimeError(
        "Repository must remain clean."
    )


# =============================================================================
# 3. RAW EXTRACTION CLOSURE GATE
# =============================================================================

banner(
    "STAGE26-4C0 :: RAW EXTRACTION CLOSURE GATE"
)


if not RAW_TIMING_MANIFEST.exists():

    raise FileNotFoundError(
        RAW_TIMING_MANIFEST
    )


raw_manifest_sha = sha256_file(
    RAW_TIMING_MANIFEST
)


print(
    "Expected Stage26-4B3 manifest SHA256:"
)

print(
    " ",
    EXPECTED_RAW_TIMING_MANIFEST_SHA256
)

print(
    "Actual:"
)

print(
    " ",
    raw_manifest_sha
)


if (
    raw_manifest_sha
    !=
    EXPECTED_RAW_TIMING_MANIFEST_SHA256
):

    raise RuntimeError(
        "Stage26-4B3 durable manifest changed."
    )


raw_manifest = json.loads(
    RAW_TIMING_MANIFEST.read_text(
        encoding="utf-8"
    )
)


if (
    raw_manifest[
        "scientific_boundaries"
    ][
        "raw_extraction_performance_measured"
    ]
    is not True
):

    raise RuntimeError(
        "Raw extraction stage not marked measured."
    )


if (
    raw_manifest[
        "scientific_boundaries"
    ][
        "release_corpus_regenerated"
    ]
    is not False
):

    raise RuntimeError(
        "Unexpected corpus-regeneration state."
    )


print(
    "\n[PASS] Raw extraction phase is durable and closed."
)


# =============================================================================
# 4. AUTHORITATIVE RELEASE CORPUS GATE
# =============================================================================

banner(
    "STAGE26-4C0 :: AUTHORITATIVE RELEASE CORPUS"
)


if not MONDAY_TAR.exists():

    raise FileNotFoundError(
        MONDAY_TAR
    )


actual_tar_size = int(
    MONDAY_TAR.stat().st_size
)


print(
    "Release TAR:"
)

print(
    " ",
    MONDAY_TAR
)

print(
    "Expected size:",
    EXPECTED_TAR_SIZE
)

print(
    "Actual size  :",
    actual_tar_size
)


if actual_tar_size != EXPECTED_TAR_SIZE:

    raise RuntimeError(
        "Monday Release TAR size changed."
    )


# Do not waste another full ~568 MiB hash pass here.
# Previous Stage26 Release audit already verified full TAR identity.
print(
    "Previously verified full TAR SHA256:"
)

print(
    " ",
    EXPECTED_TAR_SHA256
)

print(
    "Corpus regeneration from PCAP: FORBIDDEN"
)


# =============================================================================
# 5. STAGE20 ENCODER IDENTITY
# =============================================================================

banner(
    "STAGE26-4C0 :: EXACT STAGE20 ENCODER"
)


if not ENCODER.exists():

    raise FileNotFoundError(
        ENCODER
    )


encoder_sha = sha256_file(
    ENCODER
)

encoder_text = ENCODER.read_text(
    encoding="utf-8",
    errors="replace",
)


print(
    "Encoder:"
)

print(
    " ",
    ENCODER.relative_to(
        REPO
    )
)

print(
    "Bytes :",
    ENCODER.stat().st_size
)

print(
    "SHA256:"
)

print(
    " ",
    encoder_sha
)

print(
    "Lines :",
    len(
        encoder_text.splitlines()
    )
)


# =============================================================================
# 6. AST INVENTORY
# =============================================================================

banner(
    "STAGE26-4C0 :: ENCODER AST INVENTORY"
)


tree = ast.parse(
    encoder_text
)


functions = []

classes = []

assignments = []


for node in tree.body:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    ):

        functions.append(
            {
                "name":
                    node.name,

                "line":
                    int(
                        node.lineno
                    ),

                "end_line":
                    int(
                        getattr(
                            node,
                            "end_lineno",
                            node.lineno,
                        )
                    ),

                "args":
                    [
                        arg.arg
                        for arg in node.args.args
                    ],
            }
        )


    elif isinstance(
        node,
        ast.ClassDef,
    ):

        classes.append(
            {
                "name":
                    node.name,

                "line":
                    int(
                        node.lineno
                    ),

                "end_line":
                    int(
                        getattr(
                            node,
                            "end_lineno",
                            node.lineno,
                        )
                    ),
            }
        )


    elif isinstance(
        node,
        (
            ast.Assign,
            ast.AnnAssign,
        ),
    ):

        assignments.append(
            {
                "line":
                    int(
                        node.lineno
                    ),

                "text":
                    source_excerpt(
                        encoder_text,
                        node.lineno,
                        getattr(
                            node,
                            "end_lineno",
                            node.lineno,
                        ),
                    ),
            }
        )


print(
    "Top-level functions:",
    len(
        functions
    )
)


for item in functions:

    print(
        f"  line {item['line']:4d}-{item['end_line']:4d} "
        f"{item['name']}({', '.join(item['args'])})"
    )


print(
    "\nTop-level classes:",
    len(
        classes
    )
)


for item in classes:

    print(
        f"  line {item['line']:4d}-{item['end_line']:4d} "
        f"{item['name']}"
    )


# =============================================================================
# 7. KEYWORD / OPERATION SEARCH INSIDE ENCODER
# =============================================================================

banner(
    "STAGE26-4C0 :: REPRESENTATION OPERATION SEARCH"
)


patterns = {
    "64":
        r"\b64\b",

    "256":
        r"\b256\b",

    "reshape":
        r"\breshape\b",

    "zeros":
        r"\bzeros\b",

    "empty":
        r"\bempty\b",

    "uint8":
        r"\buint8\b",

    "uint16":
        r"\buint16\b",

    "float32":
        r"\bfloat32\b",

    "255":
        r"255(?:\.0)?",

    "divide":
        r"/\s*255|divide",

    "packet_lengths":
        r"packet_lengths",

    "flow_offsets":
        r"flow_offsets",

    "encoded_bytes":
        r"encoded_bytes",

    "mask":
        r"mask",

    "truncate":
        r"trunc|truncate",

    "pad":
        r"pad|padding",

    "memmap":
        r"memmap|mmap",

    "frombuffer":
        r"frombuffer",

    "torch":
        r"\btorch\b",

    "tensor":
        r"\btensor\b",
}


encoder_lines = encoder_text.splitlines()

operation_hits = {}


for label, pattern in patterns.items():

    regex = re.compile(
        pattern,
        flags=re.IGNORECASE,
    )


    hits = []


    for lineno, line in enumerate(
        encoder_lines,
        start=1,
    ):

        if regex.search(
            line
        ):

            hits.append(
                {
                    "line":
                        lineno,

                    "text":
                        line,
                }
            )


    operation_hits[
        label
    ] = hits


    print(
        f"\n[{label}] hits={len(hits)}"
    )


    for hit in hits[
        :40
    ]:

        print(
            f"  {hit['line']:5d}: "
            f"{hit['text']}"
        )


# =============================================================================
# 8. PRINT ALL FUNCTION BODIES
# =============================================================================

banner(
    "STAGE26-4C0 :: EXACT FUNCTION SOURCE"
)


for item in functions:

    print(
        "\n"
        + "-" * 122
    )

    print(
        f"{item['name']} "
        f"(lines {item['line']}-{item['end_line']})"
    )

    print(
        "-" * 122
    )

    print(
        source_excerpt(
            encoder_text,
            item[
                "line"
            ],
            item[
                "end_line"
            ],
        )
    )


# =============================================================================
# 9. REPOSITORY-WIDE REPRESENTATION SEMANTICS SEARCH
# =============================================================================

banner(
    "STAGE26-4C0 :: REPOSITORY REPRESENTATION SEMANTICS SEARCH"
)


search_terms = [
    "stage20_packet_image_encoder",
    "encoded_bytes.bin",
    "packet_lengths.npy",
    "flow_offsets.npy",
    "64x256",
    "64 x 256",
    "64, 256",
    "/ 255",
    "/255",
    "float32",
]


repo_search = {}


for term in search_terms:

    p = run(
        [
            "git",
            "grep",
            "-n",
            "-I",
            "--",
            term,
        ],
        check=False,
    )


    output = p.stdout.strip()


    rows = (
        output.splitlines()
        if output
        else []
    )


    # Keep output bounded but broad enough for audit.
    rows = rows[
        :120
    ]


    repo_search[
        term
    ] = rows


    print(
        f"\nTERM: {term!r}"
    )

    print(
        f"HITS: {len(rows)}"
    )


    for row in rows:

        print(
            " ",
            row
        )


# =============================================================================
# 10. INVENTORY HISTORICAL REPRESENTATION FILES
# =============================================================================

banner(
    "STAGE26-4C0 :: HISTORICAL STAGE20 REPRESENTATION ARTIFACTS"
)


candidate_files = []


candidate_roots = [
    REPO
    / "results"
    / "stage20_1d_representation",

    REPO
    / "results"
    / "stage20_1e_training",

    REPO
    / "scripts",
]


for root in candidate_roots:

    if not root.exists():

        continue


    for path in root.rglob(
        "*"
    ):

        if not path.is_file():

            continue


        lower = path.name.lower()


        if any(
            token in lower
            for token in [
                "representation",
                "encoder",
                "geometry",
                "image",
                "packet",
                "mask",
                "corpus",
                "cnn",
            ]
        ):

            candidate_files.append(
                path
            )


candidate_files = sorted(
    set(
        candidate_files
    )
)


print(
    "Candidate files:",
    len(
        candidate_files
    )
)


candidate_records = []


for path in candidate_files[
    :200
]:

    record = {
        "repo_relative_path":
            str(
                path.relative_to(
                    REPO
                )
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


    candidate_records.append(
        record
    )


    print(
        f"{record['size_bytes']:10,d} B  "
        f"{record['sha256']}  "
        f"{record['repo_relative_path']}"
    )


# =============================================================================
# 11. EXISTING RELEASE MEMBER HEADER / GEOMETRY AUDIT
# =============================================================================

banner(
    "STAGE26-4C0 :: RELEASE MEMBER GEOMETRY"
)


release_members = {}

arrays = {}


with tarfile.open(
    MONDAY_TAR,
    mode="r:",
) as tf:

    names = {
        member.name:
            member
        for member in tf.getmembers()
        if member.isfile()
    }


    if set(
        names
    ) != set(
        EXPECTED_MEMBERS
    ):

        raise RuntimeError(
            "Unexpected Monday Release member set."
        )


    for name, expected_sha in EXPECTED_MEMBERS.items():

        member = names[
            name
        ]


        release_members[
            name
        ] = {
            "size_bytes":
                int(
                    member.size
                ),

            "expected_sha256":
                expected_sha,
        }


        print(
            f"{name:34s} "
            f"{member.size:12,d} B "
            f"{human_bytes(member.size):>12s}"
        )


    # Only load small/medium .npy index arrays.
    for name in [
        "Monday/flow_offsets.npy",
        "Monday/labels.npy",
        "Monday/packet_lengths.npy",
    ]:

        f = tf.extractfile(
            names[
                name
            ]
        )


        if f is None:

            raise RuntimeError(
                f"Could not read {name}"
            )


        raw = f.read()


        actual_sha = sha256_bytes(
            raw
        )


        if actual_sha != EXPECTED_MEMBERS[
            name
        ]:

            raise RuntimeError(
                f"Release member SHA mismatch: {name}"
            )


        arr = np.load(
            io.BytesIO(
                raw
            ),
            allow_pickle=False,
        )


        arrays[
            name
        ] = arr


        print(
            f"\n{name}"
        )

        print(
            "  dtype:",
            arr.dtype
        )

        print(
            "  shape:",
            arr.shape
        )

        print(
            "  nbytes:",
            f"{arr.nbytes:,}"
        )


flow_offsets = arrays[
    "Monday/flow_offsets.npy"
]

labels = arrays[
    "Monday/labels.npy"
]

packet_lengths = arrays[
    "Monday/packet_lengths.npy"
]


release_geometry = {
    "flow_count":
        int(
            labels.size
        ),

    "flow_offsets_shape":
        list(
            flow_offsets.shape
        ),

    "flow_offsets_dtype":
        str(
            flow_offsets.dtype
        ),

    "labels_shape":
        list(
            labels.shape
        ),

    "labels_dtype":
        str(
            labels.dtype
        ),

    "packet_lengths_shape":
        list(
            packet_lengths.shape
        ),

    "packet_lengths_dtype":
        str(
            packet_lengths.dtype
        ),

    "packet_slots_per_flow":
        int(
            packet_lengths.shape[
                1
            ]
        ),

    "encoded_bytes_size":
        int(
            flow_offsets[
                -1
            ]
        ),
}


print(
    "\nResolved Release geometry:"
)

print(
    json.dumps(
        release_geometry,
        indent=2,
        sort_keys=True,
    )
)


# =============================================================================
# 12. STATIC DETECTION OF POSSIBLE MATERIALIZATION BOUNDARIES
# =============================================================================

banner(
    "STAGE26-4C0 :: CANDIDATE REPRESENTATION BOUNDARIES"
)


# IMPORTANT:
# These are NOT frozen claims.
# They are only candidate boundaries derived from static code inspection.
candidate_boundary_flags = {
    "encoder_mentions_64":
        len(
            operation_hits[
                "64"
            ]
        )
        > 0,

    "encoder_mentions_256":
        len(
            operation_hits[
                "256"
            ]
        )
        > 0,

    "encoder_mentions_uint8":
        len(
            operation_hits[
                "uint8"
            ]
        )
        > 0,

    "encoder_mentions_float32":
        len(
            operation_hits[
                "float32"
            ]
        )
        > 0,

    "encoder_mentions_divide_by_255":
        (
            len(
                operation_hits[
                    "divide"
                ]
            )
            > 0
        ),

    "encoder_mentions_flow_offsets":
        len(
            operation_hits[
                "flow_offsets"
            ]
        )
        > 0,

    "encoder_mentions_packet_lengths":
        len(
            operation_hits[
                "packet_lengths"
            ]
        )
        > 0,

    "encoder_mentions_encoded_bytes":
        len(
            operation_hits[
                "encoded_bytes"
            ]
        )
        > 0,
}


for key, value in candidate_boundary_flags.items():

    print(
        f"{key:46s}: {value}"
    )


print(
    "\nIMPORTANT:"
)

print(
    "  These flags are inventory observations only."
)

print(
    "  No Stage26 representation timing boundary is frozen yet."
)


# =============================================================================
# 13. WRITE LOCAL INVENTORY RECEIPT
# =============================================================================

receipt = {
    "schema":
        "stage26_4c0_representation_implementation_inventory_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4C0",

    "scientific_parent":
        EXPECTED_HEAD,

    "status":
        "INVENTORY_COMPLETE_NOT_YET_FROZEN",

    "raw_extraction": {
        "durable_manifest_sha256":
            EXPECTED_RAW_TIMING_MANIFEST_SHA256,

        "closed":
            True,
    },

    "release_corpus": {
        "tag":
            "stage20-compact-corpora-v1",

        "asset":
            "stage20-Monday-compact-corpus-v1.tar",

        "size_bytes":
            EXPECTED_TAR_SIZE,

        "sha256":
            EXPECTED_TAR_SHA256,

        "members":
            release_members,

        "geometry":
            release_geometry,

        "authoritative":
            True,

        "regeneration_from_pcap":
            False,
    },

    "stage20_encoder": {
        "repo_relative_path":
            str(
                ENCODER.relative_to(
                    REPO
                )
            ),

        "size_bytes":
            int(
                ENCODER.stat().st_size
            ),

        "sha256":
            encoder_sha,

        "functions":
            functions,

        "classes":
            classes,

        "operation_hits":
            operation_hits,
    },

    "repository_search":
        repo_search,

    "historical_representation_candidates":
        candidate_records,

    "candidate_boundary_flags":
        candidate_boundary_flags,

    "scientific_state": {
        "representation_protocol_frozen":
            False,

        "representation_timing_performed":
            False,

        "pcap_opened":
            False,

        "pcap_iterated":
            False,

        "corpus_created":
            False,

        "release_corpus_regenerated":
            False,

        "models_loaded":
            False,

        "inference_performed":
            False,

        "labels_used_for_selection":
            False,

        "Thursday_accessed":
            False,

        "Friday_accessed":
            False,

        "gpu_used":
            False,

        "git_modified":
            False,
    },

    "next_action":
        (
            "Review exact encoder/source semantics from this inventory, "
            "then freeze Stage26-4C representation profiling boundary "
            "before any representation performance measurement."
        ),
}


atomic_json(
    OUTPUT_RECEIPT,
    receipt,
)


receipt_sha = sha256_file(
    OUTPUT_RECEIPT
)


# =============================================================================
# 14. FINAL GIT AUDIT
# =============================================================================

banner(
    "STAGE26-4C0 :: FINAL GIT AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_HEAD:

    raise RuntimeError(
        "HEAD changed during representation inventory."
    )


if final_remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed during representation inventory."
    )


if final_status:

    raise RuntimeError(
        "Repository changed during representation inventory."
    )


# =============================================================================
# 15. CLOSURE
# =============================================================================

banner(
    "STAGE26-4C0 REPRESENTATION INVENTORY COMPLETE"
)


print(
    "SCIENTIFIC PARENT:"
)

print(
    " ",
    EXPECTED_HEAD
)


print(
    "\nEXACT STAGE20 ENCODER:"
)

print(
    " ",
    ENCODER.relative_to(
        REPO
    )
)

print(
    " SHA256:",
    encoder_sha
)


print(
    "\nAUTHORITATIVE RELEASE CORPUS:"
)

print(
    "  flows             :",
    f"{labels.size:,}"
)

print(
    "  packet slots/flow :",
    packet_lengths.shape[
        1
    ]
)

print(
    "  encoded bytes     :",
    f"{int(flow_offsets[-1]):,}"
)


print(
    "\nINVENTORY RECEIPT:"
)

print(
    " ",
    OUTPUT_RECEIPT
)

print(
    " SHA256:",
    receipt_sha
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  RAW EXTRACTION CLOSED              : YES"
)

print(
    "  RELEASE CORPUS AUTHORITATIVE       : YES"
)

print(
    "  RELEASE CORPUS REGENERATED         : NO"
)

print(
    "  REPRESENTATION PROTOCOL FROZEN     : NO"
)

print(
    "  REPRESENTATION TIMING              : NO"
)

print(
    "  PCAP ACCESSED                      : NO"
)

print(
    "  MODELS LOADED                      : NO"
)

print(
    "  GPU                                : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Use the exact source excerpts above to freeze Stage26-4C"
)

print(
    "  at the correct Release-corpus -> representation boundary."
)


STAGE26-4C0 :: SCIENTIFIC ANCHOR
Expected HEAD : 55aba2e1cc08385659479c6275234dc23b11d231
Local HEAD    : 55aba2e1cc08385659479c6275234dc23b11d231
origin/main   : 55aba2e1cc08385659479c6275234dc23b11d231
Repo clean    : True

STAGE26-4C0 :: RAW EXTRACTION CLOSURE GATE
Expected Stage26-4B3 manifest SHA256:
  39900c940ca25f3a541efe0dcb5f08d51408581f1f6f4b93980b093327276bc1
Actual:
  39900c940ca25f3a541efe0dcb5f08d51408581f1f6f4b93980b093327276bc1

[PASS] Raw extraction phase is durable and closed.

STAGE26-4C0 :: AUTHORITATIVE RELEASE CORPUS
Release TAR:
  /kaggle/working/stage26_deployment_profiling/release_corpora/Monday/stage20-Monday-compact-corpus-v1.tar
Expected size: 595261440
Actual size  : 595261440
Previously verified full TAR SHA256:
  4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20
Corpus regeneration from PCAP: FORBIDDEN

STAGE26-4C0 :: EXACT STAGE20 ENCODER
Encoder:
  scripts/stage20_packet_image_encoder.py
Bytes : 4883
SHA256:
  9883fe2b27020aaff707a75312

In [20]:
# =============================================================================
# STAGE26-4C1
# FREEZE RELEASE-CORPUS -> DENSE PACKET-IMAGE REPRESENTATION PROTOCOL
#
# SCIENTIFIC PARENT:
#   55aba2e1cc08385659479c6275234dc23b11d231
#
# RESOLVED COMPONENT:
#
#   EXISTING Stage20 compact Release corpus
#       encoded_bytes.bin
#       packet_lengths.npy [N,64] uint16
#       flow_offsets.npy   [N+1] uint64
#             |
#             v
#       dense image        [B,64,256] uint8
#       padding mask       [B,64,256] bool
#
# EXPLICITLY NOT INCLUDED:
#   - raw PCAP parsing
#   - flow reconstruction
#   - supervised matching
#   - packet selection
#   - header masking
#   - packet truncation/encoding into encoded_bytes.bin
#   - label access
#   - float32 / 255 scaling
#   - CNN / ViT model forward
#
# WHY /255 IS EXCLUDED:
#   It is already a model-boundary operation in the frozen inference recipes.
#   Shared representation output is uint8 image + bool padding mask.
#
# THIS CELL:
#   1. verifies the Stage26 scientific parent;
#   2. verifies Stage26-4C0 inventory and historical Stage20 code identities;
#   3. freezes the exact representation-only worker;
#   4. freezes CPU1 batch/sample/warmup/timing policy;
#   5. freezes a mandatory pre-timing equivalence gate against the historical
#      Stage20 compact loader;
#   6. writes the protocol package into Git worktree;
#   7. DOES NOT run representation reconstruction or timing.
#
# ABSOLUTE RULES:
#   - NO representation timing.
#   - NO Release TAR extraction.
#   - NO dense representation materialization.
#   - NO PCAP access.
#   - NO corpus regeneration.
#   - NO labels.
#   - NO model loading.
#   - NO inference.
#   - NO Thursday / Friday.
#   - NO GPU.
#
# NEXT AFTER THIS CELL:
#   Git-anchor this protocol package BEFORE restoration/equivalence execution.
# =============================================================================

from __future__ import annotations

import os
import json
import hashlib
import py_compile
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. FROZEN IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

REP_RUNTIME_ROOT = (
    STAGE26_ROOT
    / "representation"
)

EXPECTED_PARENT = (
    "55aba2e1cc08385659479c6275234dc23b11d231"
)


# -----------------------------------------------------------------------------
# Stage26-4C0 inventory
# -----------------------------------------------------------------------------

INVENTORY_RECEIPT = (
    REP_RUNTIME_ROOT
    / "stage26_4c0_representation_implementation_inventory.json"
)

EXPECTED_INVENTORY_SHA256 = (
    "604f0fe53c34f79e82a677f8ad57e147be0523b1530da3664b5789c8edc1c8c3"
)


# -----------------------------------------------------------------------------
# Stage26 original protocol
# -----------------------------------------------------------------------------

MEASUREMENT_PROTOCOL = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXPECTED_MEASUREMENT_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)


# -----------------------------------------------------------------------------
# Stage20 exact code
# -----------------------------------------------------------------------------

ENCODER = (
    REPO
    / "scripts"
    / "stage20_packet_image_encoder.py"
)

EXPECTED_ENCODER_SHA256 = (
    "9883fe2b27020aaff707a753123b35eb3223d21abf295d056ec233e532f94222"
)

LOADER = (
    REPO
    / "scripts"
    / "stage20_compact_corpus.py"
)

EXPECTED_LOADER_SHA256 = (
    "a1ba15881afeb1cf4de9225a06df9ae676b95f596c8ddced7734a445ba7624d0"
)

STAGE20_ARCH_LOCK = (
    REPO
    / "results"
    / "stage20_1e_training"
    / "stage20_1e0_architecture_training_protocol_lock.json"
)

EXPECTED_STAGE20_ARCH_LOCK_SHA256 = (
    "d3bba4d9d9df4432383a7f4239a8562f484cb5805b66d52794873bfaca837b3b"
)


# -----------------------------------------------------------------------------
# Authoritative Monday GitHub Release corpus
# -----------------------------------------------------------------------------

MONDAY_TAR = (
    STAGE26_ROOT
    / "release_corpora"
    / "Monday"
    / "stage20-Monday-compact-corpus-v1.tar"
)

EXPECTED_TAR_BYTES = (
    595_261_440
)

EXPECTED_TAR_SHA256 = (
    "4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20"
)

RELEASE_TAG = (
    "stage20-compact-corpora-v1"
)

RELEASE_ASSET = (
    "stage20-Monday-compact-corpus-v1.tar"
)


MEMBER_IDENTITIES = {
    "encoded_bytes.bin": {
        "tar_member":
            "Monday/encoded_bytes.bin",

        "bytes":
            522_845_159,

        "sha256":
            "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",
    },

    "flow_offsets.npy": {
        "tar_member":
            "Monday/flow_offsets.npy",

        "bytes":
            4_228_208,

        "sha256":
            "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",
    },

    "labels.npy": {
        "tar_member":
            "Monday/labels.npy",

        "bytes":
            528_637,

        "sha256":
            "48792b8d6a127b35342cb0789baa6c54396f1100a60ce7225daf08d1c3530424",
    },

    "packet_lengths.npy": {
        "tar_member":
            "Monday/packet_lengths.npy",

        "bytes":
            67_649_280,

        "sha256":
            "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
    },
}


FLOW_COUNT = (
    528_509
)

ROWS = (
    64
)

COLS = (
    256
)

ENCODED_BYTE_COUNT = (
    522_845_159
)


# =============================================================================
# 1. FROZEN REPRESENTATION EXECUTION POLICY
# =============================================================================

# Existing Stage26 reproducibility seed.
SEED = (
    26_042
)


# Use the same batch sizes already frozen for Stage26 inference.
BATCH_SIZES = [
    1,
    64,
    256,
    1024,
    8192,
]


# Nested deterministic contiguous segment.
#
# This is intentionally NOT:
#   - first-flow selected,
#   - performance-selected,
#   - label-selected,
#   - inference-result-selected.
#
# The already-frozen Stage26 seed deterministically defines the start.
MAX_BATCH_SIZE = max(
    BATCH_SIZES
)

SAMPLE_START_INDEX_ZERO_BASED = (
    SEED
    %
    (
        FLOW_COUNT
        -
        MAX_BATCH_SIZE
        +
        1
    )
)

SAMPLE_END_INDEX_ZERO_BASED_EXCLUSIVE = (
    SAMPLE_START_INDEX_ZERO_BASED
    +
    MAX_BATCH_SIZE
)


if SAMPLE_START_INDEX_ZERO_BASED != 26_042:

    raise RuntimeError(
        "Unexpected deterministic Stage26 representation sample start."
    )


# Same warm/timed iteration schedule already used for Stage26 warm inference.
ITERATION_POLICY = {
    1: {
        "warmup_runs":
            50,

        "timed_runs":
            200,

        "role":
            "LATENCY_PRIMARY",
    },

    64: {
        "warmup_runs":
            30,

        "timed_runs":
            150,

        "role":
            "MICROBATCH",
    },

    256: {
        "warmup_runs":
            20,

        "timed_runs":
            100,

        "role":
            "STANDARD_BATCH",
    },

    1024: {
        "warmup_runs":
            10,

        "timed_runs":
            50,

        "role":
            "LARGE_BATCH",
    },

    8192: {
        "warmup_runs":
            5,

        "timed_runs":
            20,

        "role":
            "CAPACITY_BATCH",
    },
}


# CPU1 only:
# - matches raw extraction component,
# - matches primary Stage26 CPU cost axis,
# - enables later CPU1 component decomposition.
CPU_AFFINITY = [
    0
]

THREAD_COUNT = (
    1
)

CONDITION_TIMEOUT_SECONDS = (
    600
)


# Same Stage26 environment gate.
ENVIRONMENT_GATE = {
    "cpu_utilization_percent_max":
        20.0,

    "available_ram_gib_min":
        8.0,

    "cpu_utilization_sampling_seconds":
        1.0,

    "retry_count":
        3,

    "retry_cooldown_seconds":
        5,

    "if_gate_never_passes":
        "INVALID_ENVIRONMENT",
}


# Mandatory equivalence test BEFORE any timing.
EQUIVALENCE_FLOW_COUNT = (
    128
)

EQUIVALENCE_START_INDEX_ZERO_BASED = (
    SAMPLE_START_INDEX_ZERO_BASED
)


# =============================================================================
# 2. OUTPUT PACKAGE
# =============================================================================

LOCK_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_4c_representation_protocol_lock"
)

LOCK_DIR = (
    REPO
    / LOCK_REL
)

WORKER_PATH = (
    LOCK_DIR
    / "stage26_representation_worker_v1.py"
)

PROTOCOL_PATH = (
    LOCK_DIR
    / "stage26_representation_protocol.json"
)

BOUNDARY_PATH = (
    LOCK_DIR
    / "stage26_representation_boundary_map.json"
)

PROVENANCE_PATH = (
    LOCK_DIR
    / "stage26_4c_upstream_provenance.json"
)

FREEZE_RECEIPT_PATH = (
    LOCK_DIR
    / "stage26_4c1_representation_protocol_freeze_receipt.json"
)

MANIFEST_PATH = (
    LOCK_DIR
    / "stage26_4c_representation_lock_manifest.json"
)


# =============================================================================
# 3. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 122
    )

    print(text)

    print(
        "=" * 122
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            p.stdout
        )

    return p


def git(*args):

    return run(
        [
            "git",
            *args,
        ]
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_text(
    path,
    text,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        f.write(
            text
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def atomic_json(
    path,
    obj,
):

    text = (
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
        )
        +
        "\n"
    )

    atomic_text(
        path,
        text,
    )


# =============================================================================
# 4. SCIENTIFIC PARENT GATE
# =============================================================================

banner(
    "STAGE26-4C1 :: SCIENTIFIC PARENT GATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-4C scientific parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before representation freeze."
    )


if status:

    raise RuntimeError(
        "Repository must be clean before representation freeze."
    )


# =============================================================================
# 5. UPSTREAM HASH GATE
# =============================================================================

banner(
    "STAGE26-4C1 :: UPSTREAM PROVENANCE GATE"
)


hash_checks = [
    (
        "Stage26-4C0 inventory",
        INVENTORY_RECEIPT,
        EXPECTED_INVENTORY_SHA256,
    ),
    (
        "Stage26 measurement protocol",
        MEASUREMENT_PROTOCOL,
        EXPECTED_MEASUREMENT_PROTOCOL_SHA256,
    ),
    (
        "Stage20 encoder",
        ENCODER,
        EXPECTED_ENCODER_SHA256,
    ),
    (
        "Stage20 compact loader",
        LOADER,
        EXPECTED_LOADER_SHA256,
    ),
    (
        "Stage20 architecture lock",
        STAGE20_ARCH_LOCK,
        EXPECTED_STAGE20_ARCH_LOCK_SHA256,
    ),
]


for label, path, expected in hash_checks:

    if not path.exists():

        raise FileNotFoundError(
            path
        )

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )

    print(
        f"{label:34s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )

    if not passed:

        raise RuntimeError(
            f"Upstream frozen hash mismatch: {label}"
        )


# =============================================================================
# 6. RELEASE SOURCE PRESENCE GATE
# =============================================================================

banner(
    "STAGE26-4C1 :: AUTHORITATIVE RELEASE SOURCE GATE"
)


if not MONDAY_TAR.exists():

    raise FileNotFoundError(
        MONDAY_TAR
    )


tar_size = int(
    MONDAY_TAR.stat().st_size
)


print(
    "Release tag :",
    RELEASE_TAG
)

print(
    "Asset       :",
    RELEASE_ASSET
)

print(
    "TAR path    :",
    MONDAY_TAR
)

print(
    "Expected B  :",
    EXPECTED_TAR_BYTES
)

print(
    "Actual B    :",
    tar_size
)

print(
    "Frozen SHA  :",
    EXPECTED_TAR_SHA256
)


if tar_size != EXPECTED_TAR_BYTES:

    raise RuntimeError(
        "Monday Release TAR size changed."
    )


print(
    "\nFull TAR is NOT redundantly re-hashed here."
)

print(
    "Release extraction performed by this cell: NO"
)

print(
    "Representation reconstruction performed : NO"
)


# =============================================================================
# 7. RESOLVED SCIENTIFIC BOUNDARY
# =============================================================================

banner(
    "STAGE26-4C1 :: RESOLVED REPRESENTATION BOUNDARY"
)


boundary = {
    "schema":
        "stage26_representation_boundary_map_v1",

    "scientific_parent":
        EXPECTED_PARENT,

    "group":
        "GROUP_B_PACKET_IMAGE",

    "component_name":
        "STAGE20_COMPACT_CORPUS_TO_DENSE_UINT8_PACKET_IMAGE_AND_BOOL_MASK",

    "authoritative_input": {
        "release_tag":
            RELEASE_TAG,

        "release_asset":
            RELEASE_ASSET,

        "release_tar_sha256":
            EXPECTED_TAR_SHA256,

        "flow_count":
            FLOW_COUNT,

        "encoded_byte_count":
            ENCODED_BYTE_COUNT,

        "files_used_by_timed_component": [
            "encoded_bytes.bin",
            "flow_offsets.npy",
            "packet_lengths.npy",
        ],

        "labels_used_by_timed_component":
            False,
    },

    "input_semantics": {
        "encoded_bytes.bin":
            (
                "Concatenated already-masked authentic retained packet bytes. "
                "Packet masking/encoding occurred upstream and is NOT repeated."
            ),

        "flow_offsets.npy":
            (
                "uint64 [N+1] byte boundaries; flow i occupies "
                "[offset[i], offset[i+1])."
            ),

        "packet_lengths.npy":
            (
                "uint16 [N,64] retained encoded packet lengths; positive "
                "lengths are contiguous from column 0; remainder zero."
            ),
    },

    "output": {
        "image_shape_per_flow":
            [
                ROWS,
                COLS,
            ],

        "image_dtype":
            "uint8",

        "padding_mask_shape_per_flow":
            [
                ROWS,
                COLS,
            ],

        "padding_mask_dtype":
            "bool",

        "batch_image_shape":
            "[B,64,256]",

        "batch_padding_mask_shape":
            "[B,64,256]",
    },

    "included_operations": [
        "read packet_lengths row",
        "read flow byte offsets",
        "allocate zero-filled uint8 batch image tensor",
        "allocate zero-filled bool batch padding-mask tensor",
        "copy encoded retained bytes into each packet-row prefix",
        "set authentic byte positions True in padding mask",
        "validate offset delta equals sum(packet_lengths) per flow",
    ],

    "excluded_operations": [
        "Release TAR unpacking/restoration",
        "raw PCAP reading",
        "packet decoding",
        "flow reconstruction",
        "supervised flow matching/filtering",
        "packet selection",
        "packet header masking",
        "packet truncation/encoding into encoded_bytes.bin",
        "labels.npy access",
        "float32 conversion",
        "division by 255",
        "torch tensor conversion",
        "CNN inference",
        "ViT inference",
        "prediction thresholding",
    ],

    "scaling_rule": {
        "representation_output_scaled":
            False,

        "float32_div255_included":
            False,

        "reason":
            (
                "Frozen Stage26 inference recipes already own model-boundary "
                "scaling. Including /255 here would double-count the ViT "
                "adapter and would move CNN model-internal work across its "
                "previously frozen inference boundary."
            ),
    },

    "claim_boundary": {
        "this_is_full_raw_flow_to_image_cost":
            False,

        "this_is_compact_corpus_dense_materialization_cost":
            True,

        "missing_upstream_packet_masking_encoding_cost_is_timed":
            False,

        "group_A_70_feature_extraction_profiled":
            False,

        "complete_end_to_end_claim_allowed":
            False,

        "component_results_are_sample_specific":
            True,
    },
}


atomic_json(
    BOUNDARY_PATH,
    boundary,
)


boundary_sha = sha256_file(
    BOUNDARY_PATH
)


print(
    "Component:"
)

print(
    " ",
    boundary[
        "component_name"
    ]
)

print(
    "\nOutput:"
)

print(
    "  image : uint8 [B,64,256]"
)

print(
    "  mask  : bool  [B,64,256]"
)

print(
    "\n/255 scaling included : NO"
)

print(
    "labels accessed       : NO"
)

print(
    "packet masking        : NO — already frozen into Release bytes"
)

print(
    "complete E2E claim    : NO"
)

print(
    "\nBoundary-map SHA256:"
)

print(
    " ",
    boundary_sha
)


# =============================================================================
# 8. WRITE REPRESENTATION-ONLY WORKER
# =============================================================================

banner(
    "STAGE26-4C1 :: WRITE FROZEN REPRESENTATION WORKER"
)


WORKER_TEXT = r'''#!/usr/bin/env python3
"""
Stage26 representation worker v1.

Scientific component:
    existing Stage20 compact corpus
        -> dense uint8 [B,64,256] images
        -> bool [B,64,256] padding masks

The worker intentionally DOES NOT:
    - access labels.npy in BENCHMARK mode,
    - apply float32 / 255 scaling,
    - load any model,
    - read any PCAP,
    - recreate any corpus.

VALIDATE_EQUIVALENCE mode may import the frozen historical Stage20 compact
loader solely to compare representation bytes/masks before timing is allowed.
"""

from __future__ import annotations

import os
import sys
import json
import time
import hashlib
import importlib.util
from pathlib import Path

import numpy as np


ROWS = 64
COLS = 256


def atomic_json(path: Path, obj) -> None:
    path = Path(path)
    tmp = Path(str(path) + ".tmp")

    with tmp.open("w", encoding="utf-8") as fh:
        json.dump(
            obj,
            fh,
            indent=2,
            sort_keys=True,
        )
        fh.write("\n")
        fh.flush()
        os.fsync(fh.fileno())

    os.replace(tmp, path)


def sha256_array(arr: np.ndarray) -> str:
    contiguous = np.ascontiguousarray(arr)
    return hashlib.sha256(
        memoryview(contiguous).cast("B")
    ).hexdigest()


class RepresentationSource:
    """
    Representation-only view of an already-restored exact Stage20 compact
    corpus. labels.npy is deliberately never opened.
    """

    def __init__(self, corpus_dir: Path):
        self.corpus_dir = Path(corpus_dir)

        self.encoded_path = (
            self.corpus_dir
            / "encoded_bytes.bin"
        )

        self.lengths_path = (
            self.corpus_dir
            / "packet_lengths.npy"
        )

        self.offsets_path = (
            self.corpus_dir
            / "flow_offsets.npy"
        )

        for path in (
            self.encoded_path,
            self.lengths_path,
            self.offsets_path,
        ):
            if not path.is_file():
                raise FileNotFoundError(path)

        self.packet_lengths = np.load(
            self.lengths_path,
            mmap_mode="r",
            allow_pickle=False,
        )

        self.flow_offsets = np.load(
            self.offsets_path,
            mmap_mode="r",
            allow_pickle=False,
        )

        self.encoded_bytes = np.memmap(
            self.encoded_path,
            dtype=np.uint8,
            mode="r",
        )

        if (
            self.packet_lengths.ndim != 2
            or
            self.packet_lengths.shape[1] != ROWS
        ):
            raise ValueError(
                "packet_lengths.npy must have shape [N,64]"
            )

        self.n = int(
            self.packet_lengths.shape[0]
        )

        if self.flow_offsets.shape != (
            self.n + 1,
        ):
            raise ValueError(
                "flow_offsets.npy must have shape [N+1]"
            )

        if self.packet_lengths.dtype != np.uint16:
            raise ValueError(
                "packet_lengths.npy must be uint16"
            )

        if self.flow_offsets.dtype != np.uint64:
            raise ValueError(
                "flow_offsets.npy must be uint64"
            )

        if int(self.flow_offsets[0]) != 0:
            raise ValueError(
                "first flow offset must be zero"
            )

        if (
            int(self.flow_offsets[-1])
            !=
            int(self.encoded_bytes.size)
        ):
            raise ValueError(
                "final flow offset does not equal encoded_bytes.bin size"
            )


    def reconstruct_representation(
        self,
        index: int,
    ) -> tuple[np.ndarray, np.ndarray]:

        index = int(index)

        if index < 0:
            index += self.n

        if (
            index < 0
            or
            index >= self.n
        ):
            raise IndexError(index)

        lengths = np.asarray(
            self.packet_lengths[index],
            dtype=np.uint16,
        )

        if np.any(
            lengths > COLS
        ):
            raise ValueError(
                "packet length exceeds frozen 256-byte width"
            )

        start = int(
            self.flow_offsets[index]
        )

        end = int(
            self.flow_offsets[index + 1]
        )

        expected = int(
            lengths.astype(
                np.uint64
            ).sum()
        )

        if end - start != expected:
            raise ValueError(
                "flow offset delta does not equal sum(packet_lengths)"
            )

        image = np.zeros(
            (
                ROWS,
                COLS,
            ),
            dtype=np.uint8,
        )

        padding_mask = np.zeros(
            (
                ROWS,
                COLS,
            ),
            dtype=np.bool_,
        )

        cursor = start

        for row_index, length in enumerate(
            lengths.tolist()
        ):

            length = int(
                length
            )

            if length == 0:
                continue

            next_cursor = (
                cursor
                +
                length
            )

            image[
                row_index,
                :length,
            ] = self.encoded_bytes[
                cursor:
                next_cursor
            ]

            padding_mask[
                row_index,
                :length,
            ] = True

            cursor = next_cursor

        if cursor != end:
            raise ValueError(
                "flow byte traversal did not terminate at expected offset"
            )

        return (
            image,
            padding_mask,
        )


    def materialize_batch(
        self,
        indices: np.ndarray,
    ) -> tuple[np.ndarray, np.ndarray]:

        indices = np.asarray(
            indices,
            dtype=np.int64,
        ).reshape(-1)

        batch_size = int(
            indices.size
        )

        images = np.zeros(
            (
                batch_size,
                ROWS,
                COLS,
            ),
            dtype=np.uint8,
        )

        masks = np.zeros(
            (
                batch_size,
                ROWS,
                COLS,
            ),
            dtype=np.bool_,
        )

        for batch_index, flow_index in enumerate(
            indices.tolist()
        ):

            image, mask = self.reconstruct_representation(
                int(flow_index)
            )

            images[
                batch_index
            ] = image

            masks[
                batch_index
            ] = mask

        return (
            images,
            masks,
        )


def load_module_from_path(
    name: str,
    path: Path,
):

    spec = importlib.util.spec_from_file_location(
        name,
        path,
    )

    if (
        spec is None
        or
        spec.loader is None
    ):
        raise RuntimeError(
            f"Could not import module: {path}"
        )

    module = importlib.util.module_from_spec(
        spec
    )

    spec.loader.exec_module(
        module
    )

    return module


def run_equivalence(
    cfg: dict,
    source: RepresentationSource,
) -> dict:

    loader_path = Path(
        cfg[
            "historical_loader_path"
        ]
    )

    loader_module = load_module_from_path(
        "stage26_frozen_stage20_compact_loader",
        loader_path,
    )

    loader_class_name = cfg[
        "historical_loader_class"
    ]

    loader_class = getattr(
        loader_module,
        loader_class_name,
    )

    historical = loader_class(
        Path(
            cfg[
                "corpus_dir"
            ]
        )
    )

    start_index = int(
        cfg[
            "start_index"
        ]
    )

    flow_count = int(
        cfg[
            "flow_count"
        ]
    )

    aggregate = hashlib.sha256()

    for flow_index in range(
        start_index,
        start_index + flow_count,
    ):

        image_new, mask_new = (
            source.reconstruct_representation(
                flow_index
            )
        )

        historical_result = historical.reconstruct(
            flow_index
        )

        if len(historical_result) < 2:
            raise RuntimeError(
                "Historical loader reconstruct() returned unexpected value."
            )

        image_old = np.asarray(
            historical_result[0]
        )

        mask_old = np.asarray(
            historical_result[1]
        )

        if not np.array_equal(
            image_new,
            image_old,
        ):
            raise RuntimeError(
                f"Image equivalence failure at flow {flow_index}"
            )

        if not np.array_equal(
            mask_new,
            mask_old,
        ):
            raise RuntimeError(
                f"Mask equivalence failure at flow {flow_index}"
            )

        aggregate.update(
            np.ascontiguousarray(
                image_new
            ).tobytes()
        )

        aggregate.update(
            np.ascontiguousarray(
                mask_new
            ).tobytes()
        )

    return {
        "status":
            "PASS",

        "mode":
            "VALIDATE_EQUIVALENCE",

        "start_index":
            start_index,

        "flow_count":
            flow_count,

        "end_index_exclusive":
            start_index + flow_count,

        "representation_fingerprint_sha256":
            aggregate.hexdigest(),

        "timing_performed":
            False,

        "labels_accessed_by_representation_worker":
            False,

        "historical_loader_used_for_validation_only":
            True,

        "gpu_used":
            False,
    }


def run_benchmark(
    cfg: dict,
    source: RepresentationSource,
) -> dict:

    start_index = int(
        cfg[
            "start_index"
        ]
    )

    batch_size = int(
        cfg[
            "batch_size"
        ]
    )

    warmup_runs = int(
        cfg[
            "warmup_runs"
        ]
    )

    timed_runs = int(
        cfg[
            "timed_runs"
        ]
    )

    indices = np.arange(
        start_index,
        start_index + batch_size,
        dtype=np.int64,
    )

    # Warm exact frozen batch before measurement.
    warm_fingerprint = None

    for _ in range(
        warmup_runs
    ):

        images, masks = source.materialize_batch(
            indices
        )

        warm_fingerprint = hashlib.sha256(
            np.ascontiguousarray(
                images
            ).tobytes()
            +
            np.ascontiguousarray(
                masks
            ).tobytes()
        ).hexdigest()


    observations = []

    reference_fingerprint = None


    for iteration in range(
        1,
        timed_runs + 1,
    ):

        start_ns = time.perf_counter_ns()

        images, masks = source.materialize_batch(
            indices
        )

        end_ns = time.perf_counter_ns()

        elapsed_ns = int(
            end_ns - start_ns
        )

        if elapsed_ns <= 0:
            raise RuntimeError(
                "Non-positive elapsed time."
            )

        elapsed_seconds = (
            elapsed_ns
            /
            1_000_000_000.0
        )

        fingerprint = hashlib.sha256(
            np.ascontiguousarray(
                images
            ).tobytes()
            +
            np.ascontiguousarray(
                masks
            ).tobytes()
        ).hexdigest()

        if reference_fingerprint is None:
            reference_fingerprint = fingerprint

        elif fingerprint != reference_fingerprint:
            raise RuntimeError(
                "Representation output changed across timed iterations."
            )

        observations.append(
            {
                "iteration_index":
                    iteration,

                "elapsed_ns":
                    elapsed_ns,

                "elapsed_seconds":
                    elapsed_seconds,

                "flows":
                    batch_size,

                "flows_per_second":
                    (
                        batch_size
                        /
                        elapsed_seconds
                    ),

                "image_output_bytes":
                    int(
                        images.nbytes
                    ),

                "mask_output_bytes":
                    int(
                        masks.nbytes
                    ),

                "total_output_bytes":
                    int(
                        images.nbytes
                        +
                        masks.nbytes
                    ),
            }
        )


    return {
        "status":
            "PASS",

        "mode":
            "BENCHMARK_TIMED",

        "start_index":
            start_index,

        "batch_size":
            batch_size,

        "end_index_exclusive":
            start_index + batch_size,

        "warmup_runs":
            warmup_runs,

        "timed_runs":
            timed_runs,

        "representation_fingerprint_sha256":
            reference_fingerprint,

        "warm_representation_fingerprint_sha256":
            warm_fingerprint,

        "observations":
            observations,

        "timing_performed":
            True,

        "labels_accessed":
            False,

        "float32_div255_performed":
            False,

        "model_loaded":
            False,

        "gpu_used":
            False,
    }


def main():

    if len(sys.argv) != 2:
        raise SystemExit(
            "usage: stage26_representation_worker_v1.py CONFIG.json"
        )

    config_path = Path(
        sys.argv[1]
    )

    cfg = json.loads(
        config_path.read_text(
            encoding="utf-8"
        )
    )

    affinity = cfg.get(
        "affinity"
    )

    if affinity is not None:

        if not hasattr(
            os,
            "sched_setaffinity",
        ):
            raise RuntimeError(
                "sched_setaffinity unavailable"
            )

        os.sched_setaffinity(
            0,
            {
                int(cpu)
                for cpu in affinity
            },
        )

    source = RepresentationSource(
        Path(
            cfg[
                "corpus_dir"
            ]
        )
    )

    expected_flow_count = int(
        cfg[
            "expected_flow_count"
        ]
    )

    if source.n != expected_flow_count:
        raise RuntimeError(
            "Restored corpus flow count mismatch."
        )

    mode = cfg[
        "mode"
    ]

    if mode == "VALIDATE_EQUIVALENCE":

        result = run_equivalence(
            cfg,
            source,
        )

    elif mode == "BENCHMARK_TIMED":

        result = run_benchmark(
            cfg,
            source,
        )

    else:

        raise RuntimeError(
            f"Unknown mode: {mode}"
        )

    result.update(
        {
            "schema":
                "stage26_representation_worker_result_v1",

            "worker_pid":
                os.getpid(),

            "flow_count_population":
                source.n,

            "image_dtype":
                "uint8",

            "mask_dtype":
                "bool",

            "image_shape_per_flow":
                [
                    ROWS,
                    COLS,
                ],

            "labels_file_opened_by_representation_source":
                False,

            "corpus_created_or_modified":
                False,

            "pcap_accessed":
                False,
        }
    )

    atomic_json(
        Path(
            cfg[
                "result_path"
            ]
        ),
        result,
    )


if __name__ == "__main__":
    main()
'''


LOCK_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


atomic_text(
    WORKER_PATH,
    WORKER_TEXT,
)


py_compile.compile(
    str(
        WORKER_PATH
    ),
    doraise=True,
)


worker_sha = sha256_file(
    WORKER_PATH
)


print(
    "Worker syntax compile: PASS"
)

print(
    "Worker:"
)

print(
    " ",
    WORKER_PATH
)

print(
    "SHA256:"
)

print(
    " ",
    worker_sha
)


# =============================================================================
# 9. FREEZE REPRESENTATION SUBPROTOCOL
# =============================================================================

banner(
    "STAGE26-4C1 :: FREEZE REPRESENTATION SUBPROTOCOL"
)


protocol = {
    "schema":
        "stage26_representation_protocol_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4C1",

    "status":
        "FROZEN_BEFORE_FIRST_REPRESENTATION_MEASUREMENT",

    "scientific_parent":
        EXPECTED_PARENT,

    "selection_independence": {
        "representation_sample_selected_using_inference_results":
            False,

        "representation_sample_selected_using_representation_timing":
            False,

        "representation_sample_selected_using_labels":
            False,

        "selection_rule":
            (
                "Use existing Stage26 seed 26042 modulo the valid "
                "start positions for the largest frozen batch."
            ),

        "seed":
            SEED,
    },

    "authoritative_source": {
        "release_tag":
            RELEASE_TAG,

        "release_asset":
            RELEASE_ASSET,

        "release_tar_bytes":
            EXPECTED_TAR_BYTES,

        "release_tar_sha256":
            EXPECTED_TAR_SHA256,

        "population_flow_count":
            FLOW_COUNT,

        "member_identities":
            MEMBER_IDENTITIES,

        "restore_before_timing":
            True,

        "restore_operation_timed":
            False,

        "restore_rule":
            (
                "Extract exact Release members byte-for-byte to an isolated "
                "Stage26 runtime directory and re-hash each member. "
                "This is restoration of the authoritative Release corpus, "
                "not corpus regeneration."
            ),
    },

    "implementation": {
        "historical_encoder_path":
            str(
                ENCODER.relative_to(
                    REPO
                )
            ),

        "historical_encoder_sha256":
            EXPECTED_ENCODER_SHA256,

        "historical_loader_path":
            str(
                LOADER.relative_to(
                    REPO
                )
            ),

        "historical_loader_sha256":
            EXPECTED_LOADER_SHA256,

        "representation_worker_path":
            str(
                WORKER_PATH.relative_to(
                    REPO
                )
            ),

        "representation_worker_sha256":
            worker_sha,

        "boundary_map_path":
            str(
                BOUNDARY_PATH.relative_to(
                    REPO
                )
            ),

        "boundary_map_sha256":
            boundary_sha,
    },

    "mandatory_pre_timing_equivalence_gate": {
        "required":
            True,

        "flow_start_index_zero_based":
            EQUIVALENCE_START_INDEX_ZERO_BASED,

        "flow_count":
            EQUIVALENCE_FLOW_COUNT,

        "comparison":
            (
                "representation-only worker image/mask must be exactly "
                "np.array_equal to the frozen Stage20 compact loader "
                "reconstruct() image/mask for every gated flow"
            ),

        "timing_during_equivalence_gate":
            False,

        "labels_may_be_used_for_selection":
            False,

        "timing_allowed_before_gate_pass":
            False,

        "gate_result_must_be_git_anchored_before_timing":
            True,
    },

    "measurement_boundary": {
        "input":
            "RESTORED_EXACT_STAGE20_COMPACT_RELEASE_FILES",

        "output":
            "UINT8_IMAGE_AND_BOOL_PADDING_MASK_BATCH",

        "image_shape":
            [
                "B",
                ROWS,
                COLS,
            ],

        "mask_shape":
            [
                "B",
                ROWS,
                COLS,
            ],

        "batch_output_allocation_included":
            True,

        "memmap_slice_reads_included":
            True,

        "Release_TAR_unpacking_included":
            False,

        "loader_initialization_included":
            False,

        "Python_process_startup_included":
            False,

        "JSON_serialization_included":
            False,

        "labels_accessed":
            False,

        "float32_conversion_included":
            False,

        "division_by_255_included":
            False,

        "model_forward_included":
            False,

        "page_cache_manipulated":
            False,

        "warm_exact_batch_before_timing":
            True,

        "cold_disk_IO_claim_allowed":
            False,

        "interpretation":
            (
                "Warm compact-corpus dense-materialization cost under the "
                "existing OS page-cache state; not a cold-storage benchmark."
            ),
    },

    "sample": {
        "population_flow_count":
            FLOW_COUNT,

        "sampling_rule":
            "DETERMINISTIC_NESTED_CONTIGUOUS_SEGMENT",

        "start_index_zero_based":
            SAMPLE_START_INDEX_ZERO_BASED,

        "largest_end_index_zero_based_exclusive":
            SAMPLE_END_INDEX_ZERO_BASED_EXCLUSIVE,

        "largest_batch_flow_count":
            MAX_BATCH_SIZE,

        "largest_batch_population_fraction":
            (
                MAX_BATCH_SIZE
                /
                FLOW_COUNT
            ),

        "batch_sizes":
            BATCH_SIZES,

        "nested_rule":
            (
                "For batch B, use flows [start, start+B). "
                "Every smaller batch is a prefix of the largest sample."
            ),

        "sample_specific_claim_required":
            True,
    },

    "hardware": {
        "mode":
            "CPU_1_PHYSICAL_CORE",

        "affinity":
            CPU_AFFINITY,

        "thread_count":
            THREAD_COUNT,

        "gpu":
            False,
    },

    "iteration_policy":
        {
            str(
                batch
            ):
                ITERATION_POLICY[
                    batch
                ]
            for batch in BATCH_SIZES
        },

    "condition_count":
        len(
            BATCH_SIZES
        ),

    "fresh_process_per_batch_condition":
        True,

    "condition_timeout_seconds":
        CONDITION_TIMEOUT_SECONDS,

    "environment_gate":
        ENVIRONMENT_GATE,

    "failure_policy": {
        "INVALID_ENVIRONMENT":
            (
                "Do not benchmark condition when the frozen environment "
                "gate fails all attempts."
            ),

        "TIMEOUT_RESOURCE_LIMIT":
            (
                "Record timeout after 600 seconds; do not reduce batch "
                "or iteration counts."
            ),

        "RESOURCE_LIMIT_OOM":
            (
                "Record resource limit; do not reduce batch size."
            ),

        "IMPLEMENTATION_FAILURE":
            (
                "Stop and perform narrow implementation diagnosis. "
                "Do not alter scientific sample or timing policy."
            ),

        "no_post_result_adaptation":
            True,
    },

    "required_raw_metrics": [
        "batch_size",
        "iteration_index",
        "elapsed_ns",
        "flows_per_second",
        "image_output_bytes",
        "mask_output_bytes",
        "total_output_bytes",
    ],

    "summary_metrics": [
        "p50_batch_latency",
        "p95_batch_latency",
        "p99_batch_latency_when_n_gte_100",
        "median_flows_per_second",
        "minimum_flows_per_second",
        "maximum_flows_per_second",
    ],

    "claim_boundary": {
        "component_name":
            (
                "Stage20 compact Release corpus -> dense uint8 image "
                "+ bool padding mask"
            ),

        "applies_to":
            "GROUP_B_PACKET_IMAGE",

        "applies_to_group_A_70_feature_models":
            False,

        "raw_flow_to_packet_image_complete":
            False,

        "packet_masking_encoding_cost_included":
            False,

        "complete_pipeline_additivity_claim_allowed":
            False,

        "representation_results_sample_specific":
            True,
    },

    "scientific_rules": {
        "release_corpus_authoritative":
            True,

        "release_corpus_regeneration_forbidden":
            True,

        "pcap_may_be_opened":
            False,

        "Thursday_may_be_accessed":
            False,

        "Friday_may_be_accessed":
            False,

        "models_may_be_loaded":
            False,

        "GPU_used":
            False,

        "first_representation_timing_allowed_only_after":
            (
                "protocol Git anchor + exact Release restoration + "
                "untimed 128-flow equivalence gate + equivalence Git anchor"
            ),
    },
}


atomic_json(
    PROTOCOL_PATH,
    protocol,
)


protocol_sha = sha256_file(
    PROTOCOL_PATH
)


print(
    "Protocol SHA256:"
)

print(
    " ",
    protocol_sha
)


print(
    "\nFrozen deterministic sample:"
)

print(
    "  population flows :",
    f"{FLOW_COUNT:,}"
)

print(
    "  start index      :",
    SAMPLE_START_INDEX_ZERO_BASED,
    "(zero-based)"
)

print(
    "  largest end      :",
    SAMPLE_END_INDEX_ZERO_BASED_EXCLUSIVE,
    "(exclusive)"
)

print(
    "  largest batch    :",
    f"{MAX_BATCH_SIZE:,}"
)

print(
    "  population frac  :",
    f"{MAX_BATCH_SIZE / FLOW_COUNT:.6%}"
)


print(
    "\nFrozen batch conditions:"
)


for batch in BATCH_SIZES:

    policy = ITERATION_POLICY[
        batch
    ]

    print(
        f"  B={batch:5d} | "
        f"flows [{SAMPLE_START_INDEX_ZERO_BASED}, "
        f"{SAMPLE_START_INDEX_ZERO_BASED + batch}) | "
        f"warm={policy['warmup_runs']:3d} | "
        f"timed={policy['timed_runs']:3d} | "
        f"{policy['role']}"
    )


# =============================================================================
# 10. UPSTREAM PROVENANCE RECORD
# =============================================================================

banner(
    "STAGE26-4C1 :: WRITE UPSTREAM PROVENANCE"
)


provenance = {
    "schema":
        "stage26_4c_upstream_provenance_v1",

    "scientific_parent":
        EXPECTED_PARENT,

    "inventory_receipt": {
        "path":
            str(
                INVENTORY_RECEIPT
            ),

        "sha256":
            EXPECTED_INVENTORY_SHA256,
    },

    "measurement_protocol": {
        "repo_relative_path":
            str(
                MEASUREMENT_PROTOCOL.relative_to(
                    REPO
                )
            ),

        "sha256":
            EXPECTED_MEASUREMENT_PROTOCOL_SHA256,
    },

    "stage20": {
        "encoder": {
            "repo_relative_path":
                str(
                    ENCODER.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_ENCODER_SHA256,
        },

        "compact_loader": {
            "repo_relative_path":
                str(
                    LOADER.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_LOADER_SHA256,
        },

        "architecture_lock": {
            "repo_relative_path":
                str(
                    STAGE20_ARCH_LOCK.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_STAGE20_ARCH_LOCK_SHA256,
        },
    },

    "release": {
        "tag":
            RELEASE_TAG,

        "asset":
            RELEASE_ASSET,

        "tar_bytes":
            EXPECTED_TAR_BYTES,

        "tar_sha256":
            EXPECTED_TAR_SHA256,

        "members":
            MEMBER_IDENTITIES,
    },

    "resolved_geometry": {
        "flow_count":
            FLOW_COUNT,

        "packet_slots_per_flow":
            ROWS,

        "bytes_per_packet_row":
            COLS,

        "encoded_byte_count":
            ENCODED_BYTE_COUNT,
    },
}


atomic_json(
    PROVENANCE_PATH,
    provenance,
)


provenance_sha = sha256_file(
    PROVENANCE_PATH
)


print(
    "Provenance SHA256:"
)

print(
    " ",
    provenance_sha
)


# =============================================================================
# 11. FREEZE RECEIPT
# =============================================================================

banner(
    "STAGE26-4C1 :: FREEZE RECEIPT"
)


freeze_receipt = {
    "schema":
        "stage26_4c1_representation_protocol_freeze_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4C1",

    "status":
        "REPRESENTATION_PROTOCOL_FROZEN_NOT_YET_EXECUTED",

    "scientific_parent":
        EXPECTED_PARENT,

    "worker_sha256":
        worker_sha,

    "protocol_sha256":
        protocol_sha,

    "boundary_map_sha256":
        boundary_sha,

    "upstream_provenance_sha256":
        provenance_sha,

    "source": {
        "release_tag":
            RELEASE_TAG,

        "release_asset":
            RELEASE_ASSET,

        "release_tar_sha256":
            EXPECTED_TAR_SHA256,

        "flow_count":
            FLOW_COUNT,
    },

    "frozen_condition": {
        "cpu":
            "CPU_1_PHYSICAL_CORE",

        "affinity":
            CPU_AFFINITY,

        "thread_count":
            THREAD_COUNT,

        "batch_sizes":
            BATCH_SIZES,

        "sample_start_index_zero_based":
            SAMPLE_START_INDEX_ZERO_BASED,

        "largest_sample_end_index_exclusive":
            SAMPLE_END_INDEX_ZERO_BASED_EXCLUSIVE,

        "seed":
            SEED,
    },

    "pre_timing_gate": {
        "equivalence_required":
            True,

        "equivalence_flow_count":
            EQUIVALENCE_FLOW_COUNT,

        "equivalence_start_index_zero_based":
            EQUIVALENCE_START_INDEX_ZERO_BASED,

        "must_be_git_anchored_before_timing":
            True,
    },

    "scientific_state": {
        "release_tar_extracted":
            False,

        "representation_materialized":
            False,

        "representation_timing_performed":
            False,

        "equivalence_gate_performed":
            False,

        "pcap_accessed":
            False,

        "release_corpus_regenerated":
            False,

        "labels_accessed":
            False,

        "models_loaded":
            False,

        "inference_performed":
            False,

        "Thursday_accessed":
            False,

        "Friday_accessed":
            False,

        "gpu_used":
            False,
    },

    "next_action":
        (
            "Git-anchor Stage26-4C1 protocol lock before restoring "
            "Release corpus members or running the untimed equivalence gate."
        ),
}


atomic_json(
    FREEZE_RECEIPT_PATH,
    freeze_receipt,
)


freeze_receipt_sha = sha256_file(
    FREEZE_RECEIPT_PATH
)


print(
    "Freeze receipt SHA256:"
)

print(
    " ",
    freeze_receipt_sha
)


# =============================================================================
# 12. BUILD LOCK MANIFEST
# =============================================================================

banner(
    "STAGE26-4C1 :: BUILD LOCK MANIFEST"
)


manifest_files = []


for path in [
    WORKER_PATH,
    PROTOCOL_PATH,
    BOUNDARY_PATH,
    PROVENANCE_PATH,
    FREEZE_RECEIPT_PATH,
]:

    manifest_files.append(
        {
            "repo_relative_path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest = {
    "schema":
        "stage26_4c_representation_lock_manifest_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        "READY_FOR_GIT_ANCHOR",

    "parent_commit":
        EXPECTED_PARENT,

    "worker_sha256":
        worker_sha,

    "representation_protocol_sha256":
        protocol_sha,

    "representation_boundary_map_sha256":
        boundary_sha,

    "upstream_provenance_sha256":
        provenance_sha,

    "freeze_receipt_sha256":
        freeze_receipt_sha,

    "file_count_excluding_manifest":
        len(
            manifest_files
        ),

    "files":
        manifest_files,
}


atomic_json(
    MANIFEST_PATH,
    manifest,
)


manifest_sha = sha256_file(
    MANIFEST_PATH
)


print(
    "Lock manifest SHA256:"
)

print(
    " ",
    manifest_sha
)


# =============================================================================
# 13. STATIC PACKAGE AUDIT
# =============================================================================

banner(
    "STAGE26-4C1 :: STATIC PACKAGE AUDIT"
)


for row in manifest_files:

    path = (
        REPO
        / row[
            "repo_relative_path"
        ]
    )

    actual_size = int(
        path.stat().st_size
    )

    actual_sha = sha256_file(
        path
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:9,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Stage26-4C1 package self-audit failed."
        )


# Ensure worker source itself does not accidentally open forbidden sources.
worker_text_lower = WORKER_TEXT.lower()


for forbidden in [
    "scapy",
    "rawpcapreader",
    "pcapng",
    "pcap_path",
    "torch",
    "cuda",
]:

    if forbidden in worker_text_lower:

        raise RuntimeError(
            f"Unexpected forbidden worker dependency/token: {forbidden}"
        )


# labels.npy may be mentioned only in explanatory text saying it is not opened.
# Verify no actual labels path construction occurs.
if (
    ' / "labels.npy"' in WORKER_TEXT
    or
    "/ 'labels.npy'" in WORKER_TEXT
):

    raise RuntimeError(
        "Representation worker unexpectedly constructs labels.npy path."
    )


print(
    "\nWorker forbidden-source static audit: PASS"
)


# =============================================================================
# 14. REPOSITORY CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-4C1 :: REPOSITORY CHANGE AUDIT"
)


repo_status = git(
    "status",
    "--porcelain",
)


print(
    repo_status
)


if not repo_status:

    raise RuntimeError(
        "Expected uncommitted Stage26-4C1 protocol package."
    )


unexpected = []


for line in repo_status.splitlines():

    rel = line[
        3:
    ]

    if not rel.startswith(
        str(
            LOCK_REL
        )
        +
        "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository modification outside Stage26-4C1:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 15. CLOSURE
# =============================================================================

banner(
    "STAGE26-4C1 REPRESENTATION PROTOCOL FREEZE COMPLETE"
)


print(
    "SCIENTIFIC PARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nAUTHORITATIVE SOURCE:"
)

print(
    "  release tag :",
    RELEASE_TAG
)

print(
    "  asset       :",
    RELEASE_ASSET
)

print(
    "  flows       :",
    f"{FLOW_COUNT:,}"
)

print(
    "  TAR SHA256  :",
    EXPECTED_TAR_SHA256
)


print(
    "\nREPRESENTATION COMPONENT:"
)

print(
    "  input  : existing masked compact-corpus bytes + lengths + offsets"
)

print(
    "  output : uint8 [B,64,256] + bool [B,64,256]"
)

print(
    "  labels : EXCLUDED"
)

print(
    "  /255   : EXCLUDED"
)

print(
    "  models : EXCLUDED"
)


print(
    "\nDETERMINISTIC SAMPLE:"
)

print(
    "  seed             :",
    SEED
)

print(
    "  start index      :",
    SAMPLE_START_INDEX_ZERO_BASED
)

print(
    "  largest end      :",
    SAMPLE_END_INDEX_ZERO_BASED_EXCLUSIVE
)

print(
    "  largest flow set :",
    f"{MAX_BATCH_SIZE:,}"
)

print(
    "  fraction Monday  :",
    f"{MAX_BATCH_SIZE / FLOW_COUNT:.6%}"
)


print(
    "\nBATCH CONDITIONS:"
)


for batch in BATCH_SIZES:

    policy = ITERATION_POLICY[
        batch
    ]

    print(
        f"  B={batch:5d}: "
        f"warm={policy['warmup_runs']:3d}, "
        f"timed={policy['timed_runs']:3d}"
    )


print(
    "\nMANDATORY PRE-TIMING GATE:"
)

print(
    "  exact 128-flow image/mask equivalence to frozen Stage20 loader"
)

print(
    "  timing during equivalence: NO"
)

print(
    "  equivalence result must be Git-anchored before first timing: YES"
)


print(
    "\nFROZEN HASHES:"
)

print(
    "  worker     :",
    worker_sha
)

print(
    "  protocol   :",
    protocol_sha
)

print(
    "  boundary   :",
    boundary_sha
)

print(
    "  provenance :",
    provenance_sha
)

print(
    "  receipt    :",
    freeze_receipt_sha
)

print(
    "  manifest   :",
    manifest_sha
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  REPRESENTATION PROTOCOL FROZEN : YES"
)

print(
    "  RELEASE TAR EXTRACTED          : NO"
)

print(
    "  REPRESENTATION MATERIALIZED    : NO"
)

print(
    "  EQUIVALENCE GATE EXECUTED      : NO"
)

print(
    "  REPRESENTATION TIMING          : NO"
)

print(
    "  PCAP ACCESSED                  : NO"
)

print(
    "  CORPUS REGENERATED             : NO"
)

print(
    "  LABELS ACCESSED                : NO"
)

print(
    "  MODELS LOADED                  : NO"
)

print(
    "  COMPLETE E2E CLAIM             : NO"
)

print(
    "  GPU                            : NO"
)


print(
    "\nNEXT:"
)

print(
    "  STAGE26-4C1-GIT — commit, push, and remotely verify this"
)

print(
    "  representation protocol lock BEFORE restoring Release members"
)

print(
    "  or executing the untimed 128-flow equivalence gate."
)


STAGE26-4C1 :: SCIENTIFIC PARENT GATE
Expected parent: 55aba2e1cc08385659479c6275234dc23b11d231
Local HEAD     : 55aba2e1cc08385659479c6275234dc23b11d231
origin/main    : 55aba2e1cc08385659479c6275234dc23b11d231
Repo clean     : True

STAGE26-4C1 :: UPSTREAM PROVENANCE GATE
Stage26-4C0 inventory              PASS 604f0fe53c34f79e82a677f8ad57e147be0523b1530da3664b5789c8edc1c8c3
Stage26 measurement protocol       PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
Stage20 encoder                    PASS 9883fe2b27020aaff707a753123b35eb3223d21abf295d056ec233e532f94222
Stage20 compact loader             PASS a1ba15881afeb1cf4de9225a06df9ae676b95f596c8ddced7734a445ba7624d0
Stage20 architecture lock          PASS d3bba4d9d9df4432383a7f4239a8562f484cb5805b66d52794873bfaca837b3b

STAGE26-4C1 :: AUTHORITATIVE RELEASE SOURCE GATE
Release tag : stage20-compact-corpora-v1
Asset       : stage20-Monday-compact-corpus-v1.tar
TAR path    : /kaggle/working/stage26_deployment_profilin

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/ids2018-validation-safe-ablation/results/stage26_deployment_profiling/stage26_4c_representation_protocol_lock/stage26_representation_boundary_map.json.tmp'

In [1]:
# =============================================================================
# STAGE26 FRESH-SESSION RECOVERY AUDIT
#
# ELECTRICITY / KAGGLE SESSION RESET OCCURRED BEFORE STAGE26-4C1 WAS PUSHED.
#
# LAST DURABLE SCIENTIFIC COMMIT:
#   55aba2e1cc08385659479c6275234dc23b11d231
#
# THIS CELL ONLY:
#   - restores/clones the Git repository if necessary;
#   - verifies origin/main against the last durable Stage26 commit;
#   - restores repository-local Git author identity;
#   - verifies Kaggle GITHUB_TOKEN exists WITHOUT printing it;
#   - inventories relevant /kaggle/input assets;
#   - checks whether transient Stage26 Release-corpus / 4C0 files survived;
#   - checks disk/RAM/CPU state.
#
# NO:
#   - PCAP access
#   - Release download
#   - corpus reconstruction
#   - representation materialization
#   - timing
#   - model loading
#   - GPU work
#   - commit
#   - push
# =============================================================================

from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

import psutil


# =============================================================================
# 0. FROZEN DURABLE STATE
# =============================================================================

REPO_URL = (
    "https://github.com/themubasshir/"
    "ids2018-validation-safe-ablation.git"
)

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_DURABLE_HEAD = (
    "55aba2e1cc08385659479c6275234dc23b11d231"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

MONDAY_RELEASE_TAR = (
    STAGE26_ROOT
    / "release_corpora"
    / "Monday"
    / "stage20-Monday-compact-corpus-v1.tar"
)

REP_INVENTORY = (
    STAGE26_ROOT
    / "representation"
    / "stage26_4c0_representation_implementation_inventory.json"
)

UNPUSHED_4C1_LOCK = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c_representation_protocol_lock"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 118
    )

    print(text)

    print(
        "=" * 118
    )


def run(
    cmd,
    *,
    cwd=None,
    check=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            + " ".join(
                map(
                    str,
                    cmd,
                )
            )
            + "\n\n"
            + p.stdout
        )

    return p


def git(*args):

    return run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        check=True,
    ).stdout.strip()


# =============================================================================
# 2. RESTORE REPOSITORY
# =============================================================================

banner(
    "STAGE26-FRESH :: REPOSITORY RESTORATION"
)


if REPO.exists():

    if not (
        REPO
        / ".git"
    ).exists():

        raise RuntimeError(
            f"{REPO} exists but is not a Git repository."
        )

    print(
        "Repository already exists:"
    )

    print(
        " ",
        REPO
    )

else:

    print(
        "Repository missing after session reset."
    )

    print(
        "Cloning fresh main..."
    )

    run(
        [
            "git",
            "clone",
            "--branch",
            "main",
            "--single-branch",
            REPO_URL,
            str(
                REPO
            ),
        ],
        cwd=Path(
            "/kaggle/working"
        ),
    )

    print(
        "Clone: PASS"
    )


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)

branch = git(
    "branch",
    "--show-current",
)


print(
    "\nBranch     :",
    branch
)

print(
    "Local HEAD :",
    head
)

print(
    "origin/main:",
    remote
)

print(
    "Expected   :",
    EXPECTED_DURABLE_HEAD
)

print(
    "Repo clean :",
    status == ""
)


if branch != "main":

    raise RuntimeError(
        "Expected main branch."
    )


if head != EXPECTED_DURABLE_HEAD:

    raise RuntimeError(
        "Local HEAD does not equal the last durable Stage26 commit."
    )


if remote != EXPECTED_DURABLE_HEAD:

    raise RuntimeError(
        "origin/main does not equal the last durable Stage26 commit."
    )


if status:

    print(
        status
    )

    raise RuntimeError(
        "Fresh repository is unexpectedly dirty."
    )


# =============================================================================
# 3. RESTORE REPOSITORY-LOCAL GIT IDENTITY
# =============================================================================

banner(
    "STAGE26-FRESH :: GIT IDENTITY"
)


author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_DURABLE_HEAD,
)

author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_DURABLE_HEAD,
)


git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


print(
    "user.name :",
    git(
        "config",
        "--local",
        "--get",
        "user.name",
    )
)

print(
    "user.email:",
    git(
        "config",
        "--local",
        "--get",
        "user.email",
    )
)


# =============================================================================
# 4. KAGGLE GITHUB SECRET
# =============================================================================

banner(
    "STAGE26-FRESH :: GITHUB SECRET"
)


try:

    from kaggle_secrets import UserSecretsClient

    github_token = UserSecretsClient().get_secret(
        "GITHUB_TOKEN"
    )

except Exception as exc:

    github_token = None

    print(
        "Secret lookup exception:",
        repr(
            exc
        )
    )


if github_token:

    print(
        "[FOUND] GITHUB_TOKEN"
    )

    print(
        "Length:",
        len(
            github_token
        )
    )

    print(
        "Value : <not printed>"
    )

else:

    print(
        "[MISSING] GITHUB_TOKEN"
    )


# Ensure token content does not remain referenced.
github_token = None


# =============================================================================
# 5. DURABLE STAGE26 CHECKPOINT INVENTORY
# =============================================================================

banner(
    "STAGE26-FRESH :: DURABLE STAGE26 CHECKPOINTS"
)


checkpoint_paths = [
    (
        "26-4B extraction protocol",
        REPO
        / "results"
        / "stage26_deployment_profiling"
        / "stage26_4b_extraction_protocol_lock",
    ),

    (
        "26-4B2 full Monday validation",
        REPO
        / "results"
        / "stage26_deployment_profiling"
        / "stage26_4b2_full_monday_validation",
    ),

    (
        "26-4B3 CPU1 extraction timing",
        REPO
        / "results"
        / "stage26_deployment_profiling"
        / "stage26_4b3_cpu1_extraction_timing",
    ),

    (
        "UNPUSHED 26-4C1 lock",
        UNPUSHED_4C1_LOCK,
    ),
]


for label, path in checkpoint_paths:

    print(
        f"{label:36s}: "
        f"{'FOUND' if path.exists() else 'ABSENT'}"
    )


if UNPUSHED_4C1_LOCK.exists():

    raise RuntimeError(
        "Unexpected Stage26-4C1 directory exists in the durable repository. "
        "We expected it to have been lost before push."
    )


# =============================================================================
# 6. TRANSIENT WORKING-STATE INVENTORY
# =============================================================================

banner(
    "STAGE26-FRESH :: TRANSIENT STATE"
)


print(
    "Stage26 working root exists :",
    STAGE26_ROOT.exists()
)

print(
    "Monday Release TAR exists   :",
    MONDAY_RELEASE_TAR.exists()
)

print(
    "4C0 inventory receipt exists :",
    REP_INVENTORY.exists()
)


if MONDAY_RELEASE_TAR.exists():

    print(
        "Monday Release TAR bytes   :",
        MONDAY_RELEASE_TAR.stat().st_size
    )


if REP_INVENTORY.exists():

    print(
        "4C0 receipt bytes           :",
        REP_INVENTORY.stat().st_size
    )


# =============================================================================
# 7. INVENTORY RELEVANT KAGGLE INPUT ASSETS
# =============================================================================

banner(
    "STAGE26-FRESH :: KAGGLE INPUT INVENTORY"
)


KAGGLE_INPUT = Path(
    "/kaggle/input"
)


keywords = (
    "monday",
    "compact",
    "stage20",
    "cic",
    "ids2018",
    "release",
)


matches = []


if KAGGLE_INPUT.exists():

    for root, dirs, files in os.walk(
        KAGGLE_INPUT
    ):

        root_path = Path(
            root
        )

        for filename in files:

            text = (
                str(
                    root_path
                    / filename
                )
            ).lower()

            if any(
                key in text
                for key in keywords
            ):

                path = (
                    root_path
                    / filename
                )

                try:

                    size = int(
                        path.stat().st_size
                    )

                except Exception:

                    size = -1

                matches.append(
                    (
                        str(
                            path
                        ),
                        size,
                    )
                )


            if len(
                matches
            ) >= 200:

                break


        if len(
            matches
        ) >= 200:

            break


print(
    "Relevant input-file matches:",
    len(
        matches
    )
)


for path, size in matches:

    print(
        f"{size:14,d} B  {path}"
    )


if not matches:

    print(
        "<none>"
    )


# =============================================================================
# 8. RUNTIME / STORAGE STATE
# =============================================================================

banner(
    "STAGE26-FRESH :: RUNTIME STATE"
)


disk = shutil.disk_usage(
    "/kaggle/working"
)

vm = psutil.virtual_memory()


print(
    "Physical CPU cores :",
    psutil.cpu_count(
        logical=False
    )
)

print(
    "Logical CPU cores  :",
    psutil.cpu_count(
        logical=True
    )
)

print(
    "Current affinity   :",
    (
        sorted(
            os.sched_getaffinity(
                0
            )
        )
        if hasattr(
            os,
            "sched_getaffinity"
        )
        else "<unavailable>"
    )
)

print(
    "Available RAM GiB  :",
    f"{vm.available / 1024**3:.3f}"
)

print(
    "Working free GiB   :",
    f"{disk.free / 1024**3:.3f}"
)

print(
    "CUDA_VISIBLE_DEVICES:",
    repr(
        os.environ.get(
            "CUDA_VISIBLE_DEVICES"
        )
    )
)


# =============================================================================
# 9. CLOSURE
# =============================================================================

banner(
    "STAGE26 FRESH-SESSION RECOVERY AUDIT COMPLETE"
)


print(
    "DURABLE SCIENTIFIC HEAD:"
)

print(
    " ",
    EXPECTED_DURABLE_HEAD
)


print(
    "\nWHAT WAS LOST:"
)

print(
    "  Stage26-4C0 transient inventory receipt may need restoration."
)

print(
    "  Stage26-4C1 uncommitted protocol package is treated as LOST."
)

print(
    "  Any hashes printed by the unpushed 4C1 attempt are NON-DURABLE"
)

print(
    "  and MUST NOT be reused as scientific anchors."
)


print(
    "\nWHAT WAS NOT LOST:"
)

print(
    "  All work through Stage26-4B3 is preserved on origin/main."
)

print(
    "  No Stage26-4C representation timing had begun."
)

print(
    "  No scientific measurement needs to be repeated."
)


print(
    "\nNEXT AFTER REVIEWING THIS OUTPUT:"
)

print(
    "  Restore the authoritative Monday Release corpus if necessary,"
)

print(
    "  recreate the untimed Stage26-4C0 inventory if necessary,"
)

print(
    "  then re-freeze Stage26-4C1 from durable HEAD 55aba2e..."
)


STAGE26-FRESH :: REPOSITORY RESTORATION
Repository missing after session reset.
Cloning fresh main...
Clone: PASS

Branch     : main
Local HEAD : 55aba2e1cc08385659479c6275234dc23b11d231
origin/main: 55aba2e1cc08385659479c6275234dc23b11d231
Expected   : 55aba2e1cc08385659479c6275234dc23b11d231
Repo clean : True

STAGE26-FRESH :: GIT IDENTITY
user.name : themubasshir
user.email: themubasshir@users.noreply.github.com

STAGE26-FRESH :: GITHUB SECRET
[FOUND] GITHUB_TOKEN
Length: 93
Value : <not printed>

STAGE26-FRESH :: DURABLE STAGE26 CHECKPOINTS
26-4B extraction protocol           : FOUND
26-4B2 full Monday validation       : FOUND
26-4B3 CPU1 extraction timing       : FOUND
UNPUSHED 26-4C1 lock                : ABSENT

STAGE26-FRESH :: TRANSIENT STATE
Stage26 working root exists : False
Monday Release TAR exists   : False
4C0 inventory receipt exists : False

STAGE26-FRESH :: KAGGLE INPUT INVENTORY
Relevant input-file matches: 1
   114,122,375 B  /kaggle/input/datasets/jmmubasshirrahm

In [2]:
# =============================================================================
# STAGE26-4C0 FRESH-SESSION RECOVERY
# RESTORE AUTHORITATIVE MONDAY RELEASE CORPUS + REBUILD UNTITMED INVENTORY
#
# DURABLE SCIENTIFIC HEAD:
#   55aba2e1cc08385659479c6275234dc23b11d231
#
# SESSION RESET LOST ONLY TRANSIENT STATE.
#
# THIS CELL:
#   1. verifies durable Git anchor;
#   2. restores ONLY the already-published Monday compact corpus from the
#      GitHub Release stage20-compact-corpora-v1;
#   3. verifies exact Release asset size + full SHA256;
#   4. verifies all four TAR member hashes and geometry;
#   5. re-audits exact Stage20 encoder / compact-loader implementation;
#   6. derives the exact historical loader class containing reconstruct();
#   7. writes a NEW transient Stage26-4C0 inventory receipt.
#
# ABSOLUTE RULES:
#   - NO PCAP access.
#   - NO corpus recreation from PCAP.
#   - NO new corpus generation.
#   - NO representation timing.
#   - NO dense image materialization.
#   - NO model loading.
#   - NO inference.
#   - NO Thursday / Friday.
#   - NO GPU work.
#   - NO Git modification.
#
# IMPORTANT:
#   Downloading/restoring the existing Release asset is NOT corpus recreation.
# =============================================================================

from __future__ import annotations

import ast
import io
import os
import json
import hashlib
import shutil
import tarfile
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import requests
import psutil


# =============================================================================
# 0. FROZEN DURABLE IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "55aba2e1cc08385659479c6275234dc23b11d231"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

RELEASE_ROOT = (
    STAGE26_ROOT
    / "release_corpora"
    / "Monday"
)

REP_ROOT = (
    STAGE26_ROOT
    / "representation"
)

MONDAY_TAR = (
    RELEASE_ROOT
    / "stage20-Monday-compact-corpus-v1.tar"
)

INVENTORY_RECEIPT = (
    REP_ROOT
    / "stage26_4c0_representation_implementation_inventory.json"
)


# -----------------------------------------------------------------------------
# GitHub Release
# -----------------------------------------------------------------------------

GITHUB_OWNER = (
    "themubasshir"
)

GITHUB_REPO = (
    "ids2018-validation-safe-ablation"
)

RELEASE_TAG = (
    "stage20-compact-corpora-v1"
)

MONDAY_ASSET_NAME = (
    "stage20-Monday-compact-corpus-v1.tar"
)

EXPECTED_TAR_SIZE = (
    595_261_440
)

EXPECTED_TAR_SHA256 = (
    "4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20"
)


# -----------------------------------------------------------------------------
# Exact existing Release members
# -----------------------------------------------------------------------------

EXPECTED_MEMBERS = {
    "Monday/encoded_bytes.bin": {
        "size":
            522_845_159,

        "sha256":
            "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",
    },

    "Monday/flow_offsets.npy": {
        "size":
            4_228_208,

        "sha256":
            "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",
    },

    "Monday/labels.npy": {
        "size":
            528_637,

        "sha256":
            "48792b8d6a127b35342cb0789baa6c54396f1100a60ce7225daf08d1c3530424",
    },

    "Monday/packet_lengths.npy": {
        "size":
            67_649_280,

        "sha256":
            "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
    },
}


# -----------------------------------------------------------------------------
# Exact durable Stage20 implementation
# -----------------------------------------------------------------------------

ENCODER = (
    REPO
    / "scripts"
    / "stage20_packet_image_encoder.py"
)

EXPECTED_ENCODER_SHA256 = (
    "9883fe2b27020aaff707a753123b35eb3223d21abf295d056ec233e532f94222"
)

LOADER = (
    REPO
    / "scripts"
    / "stage20_compact_corpus.py"
)

EXPECTED_LOADER_SHA256 = (
    "a1ba15881afeb1cf4de9225a06df9ae676b95f596c8ddced7734a445ba7624d0"
)

ARCH_LOCK = (
    REPO
    / "results"
    / "stage20_1e_training"
    / "stage20_1e0_architecture_training_protocol_lock.json"
)

EXPECTED_ARCH_LOCK_SHA256 = (
    "d3bba4d9d9df4432383a7f4239a8562f484cb5805b66d52794873bfaca837b3b"
)

MEASUREMENT_PROTOCOL = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXPECTED_MEASUREMENT_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 122
    )

    print(text)

    print(
        "=" * 122
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            p.stdout
        )

    return p


def git(*args):

    return run(
        [
            "git",
            *args,
        ]
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def human_bytes(n):

    n = float(
        n
    )

    for unit in [
        "B",
        "KiB",
        "MiB",
        "GiB",
    ]:

        if n < 1024 or unit == "GiB":

            return (
                f"{n:.3f} {unit}"
            )

        n /= 1024


# =============================================================================
# 2. DURABLE GIT GATE
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY :: DURABLE GIT GATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD :",
    EXPECTED_HEAD
)

print(
    "Local HEAD    :",
    head
)

print(
    "origin/main   :",
    remote
)

print(
    "Repo clean    :",
    status == ""
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Local HEAD differs from durable Stage26 anchor."
    )


if remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main differs from durable Stage26 anchor."
    )


if status:

    raise RuntimeError(
        "Repository must be clean before recovery."
    )


# =============================================================================
# 3. DURABLE IMPLEMENTATION HASH GATE
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY :: DURABLE IMPLEMENTATION GATE"
)


hash_checks = [
    (
        "Stage20 packet encoder",
        ENCODER,
        EXPECTED_ENCODER_SHA256,
    ),
    (
        "Stage20 compact loader",
        LOADER,
        EXPECTED_LOADER_SHA256,
    ),
    (
        "Stage20 architecture lock",
        ARCH_LOCK,
        EXPECTED_ARCH_LOCK_SHA256,
    ),
    (
        "Stage26 measurement protocol",
        MEASUREMENT_PROTOCOL,
        EXPECTED_MEASUREMENT_PROTOCOL_SHA256,
    ),
]


for label, path, expected in hash_checks:

    if not path.exists():

        raise FileNotFoundError(
            path
        )

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )

    print(
        f"{label:32s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )

    if not passed:

        raise RuntimeError(
            f"Frozen implementation mismatch: {label}"
        )


# =============================================================================
# 4. STORAGE GATE
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY :: STORAGE GATE"
)


disk = shutil.disk_usage(
    "/kaggle/working"
)

available_gib = (
    disk.free
    /
    1024**3
)


print(
    "Working free GiB:",
    f"{available_gib:.3f}"
)

print(
    "Release TAR GiB :",
    f"{EXPECTED_TAR_SIZE / 1024**3:.3f}"
)


if disk.free < (
    EXPECTED_TAR_SIZE
    +
    2 * 1024**3
):

    raise RuntimeError(
        "Insufficient safe workspace for Release restoration."
    )


RELEASE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REP_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


if INVENTORY_RECEIPT.exists():

    raise RuntimeError(
        "A Stage26-4C0 recovery inventory already exists. "
        "Do not overwrite."
    )


# =============================================================================
# 5. GITHUB RELEASE METADATA
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY :: GITHUB RELEASE METADATA"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle secret GITHUB_TOKEN unavailable."
    )


headers = {
    "Accept":
        "application/vnd.github+json",

    "Authorization":
        f"Bearer {github_token}",

    "X-GitHub-Api-Version":
        "2022-11-28",
}


release_api = (
    f"https://api.github.com/repos/"
    f"{GITHUB_OWNER}/{GITHUB_REPO}/releases/tags/{RELEASE_TAG}"
)


response = requests.get(
    release_api,
    headers=headers,
    timeout=60,
)


if response.status_code != 200:

    raise RuntimeError(
        "GitHub Release metadata request failed:\n"
        f"HTTP {response.status_code}\n"
        f"{response.text[:1000]}"
    )


release = response.json()


print(
    "Release tag:",
    release.get(
        "tag_name"
    )
)

print(
    "Release id :",
    release.get(
        "id"
    )
)

print(
    "Assets     :",
    len(
        release.get(
            "assets",
            []
        )
    )
)


assets = [
    asset
    for asset in release.get(
        "assets",
        []
    )
    if asset.get(
        "name"
    )
    ==
    MONDAY_ASSET_NAME
]


if len(
    assets
) != 1:

    raise RuntimeError(
        "Expected exactly one Monday Release asset."
    )


asset = assets[
    0
]


print(
    "\nMonday asset:"
)

print(
    "  name  :",
    asset.get(
        "name"
    )
)

print(
    "  id    :",
    asset.get(
        "id"
    )
)

print(
    "  size  :",
    asset.get(
        "size"
    )
)

print(
    "  digest:",
    asset.get(
        "digest"
    )
)


if int(
    asset[
        "size"
    ]
) != EXPECTED_TAR_SIZE:

    raise RuntimeError(
        "GitHub Release asset size differs from frozen identity."
    )


api_digest = asset.get(
    "digest"
)


if api_digest:

    expected_api_digest = (
        "sha256:"
        +
        EXPECTED_TAR_SHA256
    )

    if api_digest != expected_api_digest:

        raise RuntimeError(
            "GitHub Release digest differs from frozen Monday identity."
        )


# =============================================================================
# 6. RESTORE EXACT RELEASE TAR
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY :: RESTORE MONDAY RELEASE ASSET"
)


if MONDAY_TAR.exists():

    print(
        "Existing candidate TAR found."
    )

    print(
        "Verifying rather than downloading."
    )


    existing_size = int(
        MONDAY_TAR.stat().st_size
    )


    if existing_size != EXPECTED_TAR_SIZE:

        raise RuntimeError(
            "Existing Monday TAR has wrong size."
        )


    tar_sha = sha256_file(
        MONDAY_TAR
    )


else:

    temp_tar = Path(
        str(
            MONDAY_TAR
        )
        +
        ".download"
    )


    if temp_tar.exists():

        temp_tar.unlink()


    # GitHub asset API with octet-stream performs authenticated asset download.
    asset_download_api = (
        f"https://api.github.com/repos/"
        f"{GITHUB_OWNER}/{GITHUB_REPO}/releases/assets/"
        f"{asset['id']}"
    )


    download_headers = dict(
        headers
    )

    download_headers[
        "Accept"
    ] = "application/octet-stream"


    with requests.get(
        asset_download_api,
        headers=download_headers,
        stream=True,
        timeout=120,
        allow_redirects=True,
    ) as r:

        if r.status_code != 200:

            raise RuntimeError(
                "Monday Release download failed:\n"
                f"HTTP {r.status_code}\n"
                f"{r.text[:1000]}"
            )


        h = hashlib.sha256()

        total = 0

        next_progress = (
            128 * 1024 * 1024
        )


        with temp_tar.open(
            "wb"
        ) as f:

            for chunk in r.iter_content(
                chunk_size=8 * 1024 * 1024
            ):

                if not chunk:

                    continue

                f.write(
                    chunk
                )

                h.update(
                    chunk
                )

                total += len(
                    chunk
                )


                if total >= next_progress:

                    print(
                        "  restored:",
                        human_bytes(
                            total
                        )
                    )

                    next_progress += (
                        128 * 1024 * 1024
                    )


            f.flush()

            os.fsync(
                f.fileno()
            )


    tar_sha = h.hexdigest()


    if total != EXPECTED_TAR_SIZE:

        raise RuntimeError(
            f"Downloaded size mismatch: {total}"
        )


    if tar_sha != EXPECTED_TAR_SHA256:

        raise RuntimeError(
            "Downloaded Monday TAR SHA256 mismatch."
        )


    os.replace(
        temp_tar,
        MONDAY_TAR,
    )


print(
    "\nRestored TAR:"
)

print(
    " ",
    MONDAY_TAR
)

print(
    "Bytes :",
    MONDAY_TAR.stat().st_size
)

print(
    "SHA256:"
)

print(
    " ",
    tar_sha
)


if tar_sha != EXPECTED_TAR_SHA256:

    raise RuntimeError(
        "Final Monday TAR identity mismatch."
    )


print(
    "\n[PASS] Exact existing GitHub Release corpus restored."
)

print(
    "Corpus recreated from PCAP: NO"
)


# Token no longer needed.
github_token = None


# =============================================================================
# 7. VERIFY ALL FOUR TAR MEMBERS
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY :: TAR MEMBER BYTE VERIFICATION"
)


member_records = {}

array_payloads = {}


with tarfile.open(
    MONDAY_TAR,
    mode="r:",
) as tf:

    files = {
        member.name:
            member
        for member in tf.getmembers()
        if member.isfile()
    }


    if set(
        files
    ) != set(
        EXPECTED_MEMBERS
    ):

        print(
            "Expected members:"
        )

        for x in sorted(
            EXPECTED_MEMBERS
        ):

            print(
                " ",
                x
            )


        print(
            "\nActual members:"
        )

        for x in sorted(
            files
        ):

            print(
                " ",
                x
            )


        raise RuntimeError(
            "Monday TAR member set changed."
        )


    for member_name, frozen in EXPECTED_MEMBERS.items():

        member = files[
            member_name
        ]


        if int(
            member.size
        ) != int(
            frozen[
                "size"
            ]
        ):

            raise RuntimeError(
                f"Member size mismatch: {member_name}"
            )


        fileobj = tf.extractfile(
            member
        )


        if fileobj is None:

            raise RuntimeError(
                f"Could not read TAR member: {member_name}"
            )


        h = hashlib.sha256()

        total = 0

        chunks_for_array = (
            []
            if member_name.endswith(
                ".npy"
            )
            else None
        )


        while True:

            block = fileobj.read(
                8 * 1024 * 1024
            )

            if not block:

                break

            h.update(
                block
            )

            total += len(
                block
            )


            if chunks_for_array is not None:

                chunks_for_array.append(
                    block
                )


        digest = h.hexdigest()


        passed = (
            total
            ==
            int(
                frozen[
                    "size"
                ]
            )
            and
            digest
            ==
            frozen[
                "sha256"
            ]
        )


        print(
            f"{member_name:34s} "
            f"{'PASS' if passed else 'FAIL'} "
            f"{total:12,d} B "
            f"{digest}"
        )


        if not passed:

            raise RuntimeError(
                f"Release member identity failed: {member_name}"
            )


        member_records[
            member_name
        ] = {
            "size_bytes":
                total,

            "sha256":
                digest,
        }


        if chunks_for_array is not None:

            array_payloads[
                member_name
            ] = b"".join(
                chunks_for_array
            )


# =============================================================================
# 8. RESOLVE EXACT CORPUS GEOMETRY
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY :: RELEASE CORPUS GEOMETRY"
)


flow_offsets = np.load(
    io.BytesIO(
        array_payloads[
            "Monday/flow_offsets.npy"
        ]
    ),
    allow_pickle=False,
)

labels = np.load(
    io.BytesIO(
        array_payloads[
            "Monday/labels.npy"
        ]
    ),
    allow_pickle=False,
)

packet_lengths = np.load(
    io.BytesIO(
        array_payloads[
            "Monday/packet_lengths.npy"
        ]
    ),
    allow_pickle=False,
)


print(
    "flow_offsets:"
)

print(
    "  dtype:",
    flow_offsets.dtype
)

print(
    "  shape:",
    flow_offsets.shape
)


print(
    "\nlabels:"
)

print(
    "  dtype:",
    labels.dtype
)

print(
    "  shape:",
    labels.shape
)


print(
    "\npacket_lengths:"
)

print(
    "  dtype:",
    packet_lengths.dtype
)

print(
    "  shape:",
    packet_lengths.shape
)


FLOW_COUNT = int(
    labels.shape[
        0
    ]
)


geometry_checks = {
    "flow_count_528509":
        (
            FLOW_COUNT
            ==
            528_509
        ),

    "flow_offsets_N_plus_1":
        (
            flow_offsets.shape
            ==
            (
                FLOW_COUNT + 1,
            )
        ),

    "flow_offsets_uint64":
        (
            flow_offsets.dtype
            ==
            np.uint64
        ),

    "packet_lengths_Nx64":
        (
            packet_lengths.shape
            ==
            (
                FLOW_COUNT,
                64,
            )
        ),

    "packet_lengths_uint16":
        (
            packet_lengths.dtype
            ==
            np.uint16
        ),

    "labels_uint8":
        (
            labels.dtype
            ==
            np.uint8
        ),

    "offset_start_zero":
        (
            int(
                flow_offsets[
                    0
                ]
            )
            ==
            0
        ),

    "encoded_byte_accounting":
        (
            int(
                flow_offsets[
                    -1
                ]
            )
            ==
            EXPECTED_MEMBERS[
                "Monday/encoded_bytes.bin"
            ][
                "size"
            ]
        ),

    "packet_length_max_256":
        (
            int(
                packet_lengths.max()
            )
            <=
            256
        ),
}


for name, passed in geometry_checks.items():

    print(
        f"{name:34s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    geometry_checks.values()
):

    raise RuntimeError(
        "Recovered Release corpus geometry failed."
    )


retained_packet_counts = np.count_nonzero(
    packet_lengths,
    axis=1,
)


print(
    "\nResolved:"
)

print(
    "  flows                :",
    f"{FLOW_COUNT:,}"
)

print(
    "  packet slots / flow  :",
    packet_lengths.shape[
        1
    ]
)

print(
    "  max packet bytes     :",
    int(
        packet_lengths.max()
    )
)

print(
    "  encoded bytes        :",
    f"{int(flow_offsets[-1]):,}"
)

print(
    "  retained packets min :",
    int(
        retained_packet_counts.min()
    )
)

print(
    "  retained packets max :",
    int(
        retained_packet_counts.max()
    )
)


# =============================================================================
# 9. DERIVE HISTORICAL LOADER API FROM EXACT SOURCE
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY :: HISTORICAL LOADER API"
)


loader_text = LOADER.read_text(
    encoding="utf-8"
)

loader_tree = ast.parse(
    loader_text
)


loader_candidates = []


for node in loader_tree.body:

    if not isinstance(
        node,
        ast.ClassDef,
    ):

        continue


    method_names = {
        child.name
        for child in node.body
        if isinstance(
            child,
            ast.FunctionDef,
        )
    }


    if "reconstruct" in method_names:

        reconstruct_node = next(
            child
            for child in node.body
            if (
                isinstance(
                    child,
                    ast.FunctionDef,
                )
                and
                child.name == "reconstruct"
            )
        )


        loader_candidates.append(
            {
                "class_name":
                    node.name,

                "class_line":
                    node.lineno,

                "reconstruct_line":
                    reconstruct_node.lineno,

                "reconstruct_args":
                    [
                        arg.arg
                        for arg in reconstruct_node.args.args
                    ],
            }
        )


print(
    "Classes containing reconstruct():",
    len(
        loader_candidates
    )
)


for candidate in loader_candidates:

    print(
        " ",
        candidate
    )


if len(
    loader_candidates
) != 1:

    raise RuntimeError(
        "Could not uniquely derive historical compact-loader class."
    )


HISTORICAL_LOADER_CLASS = loader_candidates[
    0
][
    "class_name"
]


print(
    "\nDerived historical loader class:"
)

print(
    " ",
    HISTORICAL_LOADER_CLASS
)


# =============================================================================
# 10. STATIC REPRESENTATION SEMANTICS
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY :: REPRESENTATION SEMANTICS"
)


encoder_text = ENCODER.read_text(
    encoding="utf-8"
)


semantic_checks = {
    "MAX_PACKETS_64":
        (
            re_search := (
                "MAX_PACKETS = 64"
                in
                encoder_text
            )
        ),

    "MAX_BYTES_PER_PACKET_256":
        (
            "MAX_BYTES_PER_PACKET = 256"
            in
            encoder_text
        ),

    "uint8_representation":
        (
            "np.uint8"
            in
            encoder_text
        ),

    "float32_model_scaling_exists":
        (
            "np.float32"
            in
            encoder_text
        ),

    "divide_by_255_exists":
        (
            "/ 255"
            in
            encoder_text
            or
            "/255"
            in
            encoder_text
        ),

    "historical_loader_reconstruct_exists":
        (
            "def reconstruct"
            in
            loader_text
        ),

    "loader_uses_encoded_bytes":
        (
            "encoded_bytes"
            in
            loader_text
        ),

    "loader_uses_packet_lengths":
        (
            "packet_lengths"
            in
            loader_text
        ),

    "loader_uses_flow_offsets":
        (
            "flow_offsets"
            in
            loader_text
        ),
}


for name, passed in semantic_checks.items():

    print(
        f"{name:38s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    semantic_checks.values()
):

    raise RuntimeError(
        "Stage20 representation semantics no longer match expected implementation."
    )


print(
    "\nResolved representation boundary:"
)

print(
    "  input : existing masked compact-corpus bytes + offsets + lengths"
)

print(
    "  dense : uint8 [64,256]"
)

print(
    "  mask  : historical loader reconstruct() padding mask"
)

print(
    "  /255  : separate model-boundary operation"
)

print(
    "  raw packet masking/encoding timing: NOT represented by this corpus"
)


# =============================================================================
# 11. WRITE NEW TRANSIENT 4C0 INVENTORY RECEIPT
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY :: WRITE INVENTORY RECEIPT"
)


inventory = {
    "schema":
        "stage26_4c0_representation_implementation_inventory_v2_fresh_session",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4C0-RECOVERY",

    "status":
        "PASS_INVENTORY_REBUILT_AFTER_SESSION_RESET",

    "scientific_parent":
        EXPECTED_HEAD,

    "session_recovery": {
        "previous_transient_4c0_lost":
            True,

        "previous_unpushed_4c1_treated_as_lost":
            True,

        "scientific_measurement_repeated":
            False,

        "reason":
            "Kaggle session reset before Stage26-4C1 Git anchor.",
    },

    "authoritative_release": {
        "repository":
            f"{GITHUB_OWNER}/{GITHUB_REPO}",

        "tag":
            RELEASE_TAG,

        "asset":
            MONDAY_ASSET_NAME,

        "size_bytes":
            EXPECTED_TAR_SIZE,

        "sha256":
            EXPECTED_TAR_SHA256,

        "restored_from_existing_release":
            True,

        "recreated_from_pcap":
            False,

        "members":
            member_records,
    },

    "geometry": {
        "flow_count":
            FLOW_COUNT,

        "flow_offsets_shape":
            list(
                flow_offsets.shape
            ),

        "flow_offsets_dtype":
            str(
                flow_offsets.dtype
            ),

        "packet_lengths_shape":
            list(
                packet_lengths.shape
            ),

        "packet_lengths_dtype":
            str(
                packet_lengths.dtype
            ),

        "labels_shape":
            list(
                labels.shape
            ),

        "labels_dtype":
            str(
                labels.dtype
            ),

        "packet_slots_per_flow":
            64,

        "max_encoded_bytes_per_packet":
            256,

        "encoded_byte_count":
            int(
                flow_offsets[
                    -1
                ]
            ),
    },

    "stage20_implementation": {
        "encoder": {
            "repo_relative_path":
                str(
                    ENCODER.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_ENCODER_SHA256,
        },

        "compact_loader": {
            "repo_relative_path":
                str(
                    LOADER.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_LOADER_SHA256,

            "derived_reconstruct_class":
                HISTORICAL_LOADER_CLASS,
        },

        "architecture_lock_sha256":
            EXPECTED_ARCH_LOCK_SHA256,

        "measurement_protocol_sha256":
            EXPECTED_MEASUREMENT_PROTOCOL_SHA256,
    },

    "resolved_boundary": {
        "component":
            (
                "existing Stage20 compact Release corpus "
                "-> dense packet-image representation"
            ),

        "input_files": [
            "encoded_bytes.bin",
            "flow_offsets.npy",
            "packet_lengths.npy",
        ],

        "labels_required":
            False,

        "dense_image_shape_per_flow": [
            64,
            256,
        ],

        "dense_image_dtype":
            "uint8",

        "float32_div255_part_of_this_component":
            False,

        "raw_packet_header_masking_part_of_this_component":
            False,

        "raw_packet_truncation_encoding_part_of_this_component":
            False,

        "complete_raw_flow_to_model_E2E_claim_allowed":
            False,
    },

    "scientific_state": {
        "pcap_accessed":
            False,

        "pcap_iterated":
            False,

        "release_asset_downloaded":
            True,

        "release_tar_unpacked_to_new_corpus":
            False,

        "release_corpus_regenerated":
            False,

        "representation_materialized":
            False,

        "representation_timing_performed":
            False,

        "models_loaded":
            False,

        "inference_performed":
            False,

        "Thursday_accessed":
            False,

        "Friday_accessed":
            False,

        "gpu_used":
            False,

        "git_modified":
            False,
    },

    "next_action":
        (
            "Re-freeze Stage26-4C1 representation protocol from durable "
            "scientific parent using this fresh-session inventory receipt, "
            "then Git-anchor it before any equivalence execution or timing."
        ),
}


atomic_json(
    INVENTORY_RECEIPT,
    inventory,
)


inventory_sha = sha256_file(
    INVENTORY_RECEIPT
)


print(
    "Inventory receipt:"
)

print(
    " ",
    INVENTORY_RECEIPT
)

print(
    "SHA256:"
)

print(
    " ",
    inventory_sha
)


# =============================================================================
# 12. FINAL GIT / RUNTIME AUDIT
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY :: FINAL AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)

print(
    "Available RAM GiB:",
    f"{psutil.virtual_memory().available / 1024**3:.3f}"
)

print(
    "Working free GiB :",
    f"{shutil.disk_usage('/kaggle/working').free / 1024**3:.3f}"
)


if final_head != EXPECTED_HEAD:

    raise RuntimeError(
        "HEAD changed during 4C0 recovery."
    )


if final_remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed during 4C0 recovery."
    )


if final_status:

    raise RuntimeError(
        "Repository changed during 4C0 recovery."
    )


# =============================================================================
# 13. CLOSURE
# =============================================================================

banner(
    "STAGE26-4C0 FRESH-SESSION RECOVERY COMPLETE"
)


print(
    "DURABLE SCIENTIFIC HEAD:"
)

print(
    " ",
    EXPECTED_HEAD
)


print(
    "\nAUTHORITATIVE MONDAY RELEASE:"
)

print(
    "  tag       :",
    RELEASE_TAG
)

print(
    "  asset     :",
    MONDAY_ASSET_NAME
)

print(
    "  bytes     :",
    f"{EXPECTED_TAR_SIZE:,}"
)

print(
    "  SHA256    :",
    EXPECTED_TAR_SHA256
)

print(
    "  flows     :",
    f"{FLOW_COUNT:,}"
)

print(
    "  recreated : NO"
)


print(
    "\nHISTORICAL IMPLEMENTATION:"
)

print(
    "  encoder SHA :",
    EXPECTED_ENCODER_SHA256
)

print(
    "  loader SHA  :",
    EXPECTED_LOADER_SHA256
)

print(
    "  loader class:",
    HISTORICAL_LOADER_CLASS
)


print(
    "\nRESOLVED REPRESENTATION:"
)

print(
    "  compact bytes + offsets + lengths"
)

print(
    "      -> uint8 64x256 dense image + historical padding mask"
)

print(
    "  /255 scaling                  : EXCLUDED"
)

print(
    "  raw masking/encoding          : EXCLUDED"
)

print(
    "  labels                        : NOT NEEDED FOR REPRESENTATION"
)

print(
    "  complete E2E claim            : BLOCKED"
)


print(
    "\nNEW 4C0 INVENTORY SHA256:"
)

print(
    " ",
    inventory_sha
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  RELEASE CORPUS RESTORED        : YES"
)

print(
    "  RESTORED FROM GITHUB RELEASE   : YES"
)

print(
    "  RECREATED FROM PCAP            : NO"
)

print(
    "  REPRESENTATION MATERIALIZED    : NO"
)

print(
    "  REPRESENTATION TIMING          : NO"
)

print(
    "  PCAP ACCESSED                  : NO"
)

print(
    "  MODELS LOADED                  : NO"
)

print(
    "  GPU                            : NO"
)

print(
    "  GIT MODIFIED                   : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Re-freeze Stage26-4C1 using THIS new inventory SHA,"
)

print(
    "  then commit/push the protocol lock immediately."
)


STAGE26-4C0-RECOVERY :: DURABLE GIT GATE
Expected HEAD : 55aba2e1cc08385659479c6275234dc23b11d231
Local HEAD    : 55aba2e1cc08385659479c6275234dc23b11d231
origin/main   : 55aba2e1cc08385659479c6275234dc23b11d231
Repo clean    : True

STAGE26-4C0-RECOVERY :: DURABLE IMPLEMENTATION GATE
Stage20 packet encoder           PASS 9883fe2b27020aaff707a753123b35eb3223d21abf295d056ec233e532f94222
Stage20 compact loader           PASS a1ba15881afeb1cf4de9225a06df9ae676b95f596c8ddced7734a445ba7624d0
Stage20 architecture lock        PASS d3bba4d9d9df4432383a7f4239a8562f484cb5805b66d52794873bfaca837b3b
Stage26 measurement protocol     PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625

STAGE26-4C0-RECOVERY :: STORAGE GATE
Working free GiB: 17.310
Release TAR GiB : 0.554

STAGE26-4C0-RECOVERY :: GITHUB RELEASE METADATA
Release tag: stage20-compact-corpora-v1
Release id : 371092195
Assets     : 10

Monday asset:
  name  : stage20-Monday-compact-corpus-v1.tar
  id    : 515837749
  si

RuntimeError: Stage20 representation semantics no longer match expected implementation.

In [3]:
# =============================================================================
# STAGE26-4C0-RECOVERY-R
# NARROW RECOVERY FROM BRITTLE LEXICAL CONSTANT CHECK
#
# PREVIOUS CELL ALREADY PASSED:
#   - durable Git anchor
#   - exact Stage20 implementation hashes
#   - GitHub Release metadata
#   - complete Monday TAR download
#   - full TAR SHA256
#   - all four TAR member SHA256 checks
#   - corpus geometry
#   - historical loader API derivation:
#         Stage20CompactCorpus.reconstruct(self, index)
#
# PREVIOUS CELL FAILED ONLY BECAUSE:
#   literal string checks such as:
#       "MAX_PACKETS = 64"
#   are formatting-sensitive.
#
# THIS CELL:
#   - DOES NOT download the Release again;
#   - DOES NOT hash encoded_bytes.bin again;
#   - resolves constants semantically from Python AST;
#   - rechecks only the small .npy geometry members;
#   - confirms scale_for_model semantics structurally;
#   - writes the fresh-session Stage26-4C0 inventory receipt.
#
# NO:
#   - PCAP access
#   - corpus reconstruction
#   - representation materialization
#   - representation timing
#   - model loading
#   - inference
#   - GPU
#   - Git modification
# =============================================================================

from __future__ import annotations

import ast
import io
import os
import re
import json
import hashlib
import tarfile
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np


# =============================================================================
# 0. FROZEN IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "55aba2e1cc08385659479c6275234dc23b11d231"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

REP_ROOT = (
    STAGE26_ROOT
    / "representation"
)

REP_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

INVENTORY_RECEIPT = (
    REP_ROOT
    / "stage26_4c0_representation_implementation_inventory.json"
)


MONDAY_TAR = (
    STAGE26_ROOT
    / "release_corpora"
    / "Monday"
    / "stage20-Monday-compact-corpus-v1.tar"
)

EXPECTED_TAR_SIZE = (
    595_261_440
)

EXPECTED_TAR_SHA256 = (
    "4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20"
)


EXPECTED_MEMBERS = {
    "Monday/encoded_bytes.bin": {
        "size":
            522_845_159,

        "sha256":
            "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",
    },

    "Monday/flow_offsets.npy": {
        "size":
            4_228_208,

        "sha256":
            "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",
    },

    "Monday/labels.npy": {
        "size":
            528_637,

        "sha256":
            "48792b8d6a127b35342cb0789baa6c54396f1100a60ce7225daf08d1c3530424",
    },

    "Monday/packet_lengths.npy": {
        "size":
            67_649_280,

        "sha256":
            "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
    },
}


ENCODER = (
    REPO
    / "scripts"
    / "stage20_packet_image_encoder.py"
)

EXPECTED_ENCODER_SHA256 = (
    "9883fe2b27020aaff707a753123b35eb3223d21abf295d056ec233e532f94222"
)


LOADER = (
    REPO
    / "scripts"
    / "stage20_compact_corpus.py"
)

EXPECTED_LOADER_SHA256 = (
    "a1ba15881afeb1cf4de9225a06df9ae676b95f596c8ddced7734a445ba7624d0"
)


ARCH_LOCK = (
    REPO
    / "results"
    / "stage20_1e_training"
    / "stage20_1e0_architecture_training_protocol_lock.json"
)

EXPECTED_ARCH_LOCK_SHA256 = (
    "d3bba4d9d9df4432383a7f4239a8562f484cb5805b66d52794873bfaca837b3b"
)


MEASUREMENT_PROTOCOL = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXPECTED_MEASUREMENT_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 122
    )

    print(text)

    print(
        "=" * 122
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            p.stdout
        )

    return p


def git(*args):

    return run(
        [
            "git",
            *args,
        ]
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def source_segment(
    source,
    node,
):

    segment = ast.get_source_segment(
        source,
        node,
    )

    return (
        segment
        if segment is not None
        else "<source unavailable>"
    )


def assignment_name(node):

    if isinstance(
        node,
        ast.Assign,
    ):

        if len(
            node.targets
        ) != 1:

            return None

        target = node.targets[
            0
        ]

    elif isinstance(
        node,
        ast.AnnAssign,
    ):

        target = node.target

    else:

        return None


    if isinstance(
        target,
        ast.Name,
    ):

        return target.id


    return None


# =============================================================================
# 2. DURABLE STATE GATE
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-R :: DURABLE STATE"
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD :",
    EXPECTED_HEAD
)

print(
    "Local HEAD    :",
    head
)

print(
    "origin/main   :",
    remote
)

print(
    "Repo clean    :",
    status == ""
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected HEAD."
    )


if remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed."
    )


if status:

    raise RuntimeError(
        "Repository must remain clean."
    )


if INVENTORY_RECEIPT.exists():

    raise RuntimeError(
        "4C0 inventory receipt already exists; do not overwrite."
    )


# =============================================================================
# 3. NARROW IDENTITY GATE
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-R :: IDENTITY GATE"
)


hash_checks = [
    (
        "Stage20 encoder",
        ENCODER,
        EXPECTED_ENCODER_SHA256,
    ),
    (
        "Stage20 compact loader",
        LOADER,
        EXPECTED_LOADER_SHA256,
    ),
    (
        "Stage20 architecture lock",
        ARCH_LOCK,
        EXPECTED_ARCH_LOCK_SHA256,
    ),
    (
        "Stage26 measurement protocol",
        MEASUREMENT_PROTOCOL,
        EXPECTED_MEASUREMENT_PROTOCOL_SHA256,
    ),
]


for label, path, expected in hash_checks:

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )

    print(
        f"{label:32s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )

    if not passed:

        raise RuntimeError(
            f"Identity changed: {label}"
        )


if not MONDAY_TAR.exists():

    raise FileNotFoundError(
        MONDAY_TAR
    )


tar_size = int(
    MONDAY_TAR.stat().st_size
)


print(
    "\nMonday TAR size:",
    tar_size
)

print(
    "Expected       :",
    EXPECTED_TAR_SIZE
)

print(
    "Previously verified full SHA256:"
)

print(
    " ",
    EXPECTED_TAR_SHA256
)


if tar_size != EXPECTED_TAR_SIZE:

    raise RuntimeError(
        "Restored Monday TAR size changed."
    )


print(
    "\nFull TAR / encoded_bytes hashes are NOT repeated."
)

print(
    "They already passed in the immediately preceding recovery cell."
)


# =============================================================================
# 4. PARSE EXACT ENCODER AST
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-R :: ENCODER AST CONSTANT RESOLUTION"
)


encoder_text = ENCODER.read_text(
    encoding="utf-8"
)

encoder_tree = ast.parse(
    encoder_text
)


module_assignments = {}


for node in encoder_tree.body:

    name = assignment_name(
        node
    )

    if name is None:

        continue


    value_node = (
        node.value
        if hasattr(
            node,
            "value"
        )
        else None
    )


    literal_value = None

    literal_ok = False


    if value_node is not None:

        try:

            literal_value = ast.literal_eval(
                value_node
            )

            literal_ok = True

        except Exception:

            pass


    module_assignments[
        name
    ] = {
        "line":
            int(
                node.lineno
            ),

        "literal_ok":
            literal_ok,

        "literal_value":
            literal_value,

        "source":
            source_segment(
                encoder_text,
                node,
            ),
    }


# Print all relevant packet/byte/image constants for provenance.
interesting = {
    name:
        info
    for name, info in module_assignments.items()
    if any(
        token in name.upper()
        for token in [
            "PACKET",
            "BYTE",
            "ROW",
            "COL",
            "IMAGE",
            "WIDTH",
            "HEIGHT",
        ]
    )
}


print(
    "Relevant module assignments:"
)


for name in sorted(
    interesting
):

    info = interesting[
        name
    ]

    print(
        f"  line {info['line']:4d} | "
        f"{name:30s} | "
        f"literal={info['literal_value']!r}"
    )

    print(
        "      ",
        info[
            "source"
        ].replace(
            "\n",
            " "
        )
    )


# =============================================================================
# 5. SEMANTIC CONSTANT RESOLUTION
# =============================================================================

def resolve_named_integer(
    preferred_names,
    expected_value,
):

    matches = []


    # First: exact preferred names.
    for name in preferred_names:

        info = module_assignments.get(
            name
        )

        if (
            info is not None
            and
            info[
                "literal_ok"
            ]
            and
            isinstance(
                info[
                    "literal_value"
                ],
                int,
            )
            and
            int(
                info[
                    "literal_value"
                ]
            )
            ==
            expected_value
        ):

            matches.append(
                {
                    "name":
                        name,

                    **info,
                }
            )


    if matches:

        return matches[
            0
        ]


    # Second: semantic fallback among packet/byte-related module constants.
    for name, info in interesting.items():

        if not info[
            "literal_ok"
        ]:

            continue


        value = info[
            "literal_value"
        ]


        if (
            isinstance(
                value,
                int,
            )
            and
            int(
                value
            )
            ==
            expected_value
        ):

            matches.append(
                {
                    "name":
                        name,

                    **info,
                }
            )


    if len(
        matches
    ) == 1:

        return matches[
            0
        ]


    print(
        f"\nCould not uniquely resolve expected integer {expected_value}."
    )

    print(
        "Candidate matches:"
    )


    for item in matches:

        print(
            " ",
            item
        )


    raise RuntimeError(
        f"Could not uniquely resolve frozen encoder constant {expected_value}."
    )


max_packets_resolution = resolve_named_integer(
    [
        "MAX_PACKETS",
        "MAX_PACKETS_PER_FLOW",
        "PACKETS_PER_FLOW",
        "IMAGE_ROWS",
        "ROWS",
    ],
    64,
)


max_bytes_resolution = resolve_named_integer(
    [
        "MAX_BYTES_PER_PACKET",
        "BYTES_PER_PACKET",
        "PACKET_BYTES",
        "IMAGE_COLS",
        "COLS",
    ],
    256,
)


print(
    "\nResolved packet-row count:"
)

print(
    "  name  :",
    max_packets_resolution[
        "name"
    ]
)

print(
    "  value :",
    max_packets_resolution[
        "literal_value"
    ]
)

print(
    "  line  :",
    max_packets_resolution[
        "line"
    ]
)

print(
    "  source:",
    max_packets_resolution[
        "source"
    ]
)


print(
    "\nResolved bytes-per-row:"
)

print(
    "  name  :",
    max_bytes_resolution[
        "name"
    ]
)

print(
    "  value :",
    max_bytes_resolution[
        "literal_value"
    ]
)

print(
    "  line  :",
    max_bytes_resolution[
        "line"
    ]
)

print(
    "  source:",
    max_bytes_resolution[
        "source"
    ]
)


MAX_PACKETS = int(
    max_packets_resolution[
        "literal_value"
    ]
)

MAX_BYTES_PER_PACKET = int(
    max_bytes_resolution[
        "literal_value"
    ]
)


if MAX_PACKETS != 64:

    raise RuntimeError(
        "Resolved packet-row count is not 64."
    )


if MAX_BYTES_PER_PACKET != 256:

    raise RuntimeError(
        "Resolved packet-byte width is not 256."
    )


# =============================================================================
# 6. RESOLVE scale_for_model STRUCTURALLY
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-R :: MODEL-SCALING BOUNDARY"
)


scale_functions = [
    node
    for node in encoder_tree.body
    if (
        isinstance(
            node,
            ast.FunctionDef,
        )
        and
        node.name == "scale_for_model"
    )
]


if len(
    scale_functions
) != 1:

    raise RuntimeError(
        "Expected exactly one scale_for_model() function."
    )


scale_node = scale_functions[
    0
]

scale_source = source_segment(
    encoder_text,
    scale_node,
)


print(
    scale_source
)


has_float32 = False

has_255_division = False


for node in ast.walk(
    scale_node
):

    # np.float32 or bare float32
    if isinstance(
        node,
        ast.Attribute,
    ):

        if node.attr == "float32":

            has_float32 = True


    elif isinstance(
        node,
        ast.Name,
    ):

        if node.id == "float32":

            has_float32 = True


    if isinstance(
        node,
        ast.BinOp,
    ) and isinstance(
        node.op,
        ast.Div,
    ):

        try:

            divisor = ast.literal_eval(
                node.right
            )

            if float(
                divisor
            ) == 255.0:

                has_255_division = True

        except Exception:

            pass


print(
    "\nfloat32 conversion detected:",
    has_float32
)

print(
    "division by 255 detected    :",
    has_255_division
)


if not has_float32:

    raise RuntimeError(
        "Could not structurally confirm float32 model scaling."
    )


if not has_255_division:

    raise RuntimeError(
        "Could not structurally confirm /255 model scaling."
    )


# =============================================================================
# 7. HISTORICAL COMPACT LOADER API
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-R :: HISTORICAL LOADER API"
)


loader_text = LOADER.read_text(
    encoding="utf-8"
)

loader_tree = ast.parse(
    loader_text
)


loader_candidates = []


for node in loader_tree.body:

    if not isinstance(
        node,
        ast.ClassDef,
    ):

        continue


    for child in node.body:

        if (
            isinstance(
                child,
                ast.FunctionDef,
            )
            and
            child.name
            ==
            "reconstruct"
        ):

            loader_candidates.append(
                {
                    "class_name":
                        node.name,

                    "class_line":
                        int(
                            node.lineno
                        ),

                    "reconstruct_line":
                        int(
                            child.lineno
                        ),

                    "reconstruct_args":
                        [
                            arg.arg
                            for arg in child.args.args
                        ],

                    "reconstruct_source":
                        source_segment(
                            loader_text,
                            child,
                        ),
                }
            )


if len(
    loader_candidates
) != 1:

    raise RuntimeError(
        "Historical reconstruct() class is not unique."
    )


loader_api = loader_candidates[
    0
]

HISTORICAL_LOADER_CLASS = loader_api[
    "class_name"
]


print(
    "Class:",
    HISTORICAL_LOADER_CLASS
)

print(
    "Args :",
    loader_api[
        "reconstruct_args"
    ]
)

print(
    "Line :",
    loader_api[
        "reconstruct_line"
    ]
)


if loader_api[
    "reconstruct_args"
][:2] != [
    "self",
    "index",
]:

    raise RuntimeError(
        "Unexpected historical reconstruct() signature."
    )


# =============================================================================
# 8. RECHECK SMALL RELEASE INDEX ARRAYS ONLY
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-R :: SMALL RELEASE GEOMETRY RECHECK"
)


small_members = [
    "Monday/flow_offsets.npy",
    "Monday/labels.npy",
    "Monday/packet_lengths.npy",
]


array_payloads = {}


with tarfile.open(
    MONDAY_TAR,
    mode="r:",
) as tf:

    members = {
        member.name:
            member
        for member in tf.getmembers()
        if member.isfile()
    }


    if set(
        members
    ) != set(
        EXPECTED_MEMBERS
    ):

        raise RuntimeError(
            "Monday TAR member set changed."
        )


    for member_name in small_members:

        member = members[
            member_name
        ]

        f = tf.extractfile(
            member
        )


        if f is None:

            raise RuntimeError(
                f"Unable to read {member_name}"
            )


        raw = f.read()

        digest = sha256_bytes(
            raw
        )

        expected = EXPECTED_MEMBERS[
            member_name
        ]


        passed = (
            len(
                raw
            )
            ==
            expected[
                "size"
            ]
            and
            digest
            ==
            expected[
                "sha256"
            ]
        )


        print(
            f"{member_name:34s} "
            f"{'PASS' if passed else 'FAIL'} "
            f"{len(raw):12,d} B "
            f"{digest}"
        )


        if not passed:

            raise RuntimeError(
                f"Small member identity failed: {member_name}"
            )


        array_payloads[
            member_name
        ] = raw


flow_offsets = np.load(
    io.BytesIO(
        array_payloads[
            "Monday/flow_offsets.npy"
        ]
    ),
    allow_pickle=False,
)

labels = np.load(
    io.BytesIO(
        array_payloads[
            "Monday/labels.npy"
        ]
    ),
    allow_pickle=False,
)

packet_lengths = np.load(
    io.BytesIO(
        array_payloads[
            "Monday/packet_lengths.npy"
        ]
    ),
    allow_pickle=False,
)


FLOW_COUNT = int(
    labels.size
)


geometry_checks = {
    "flows_528509":
        (
            FLOW_COUNT
            ==
            528_509
        ),

    "offsets_shape_N_plus_1":
        (
            flow_offsets.shape
            ==
            (
                FLOW_COUNT + 1,
            )
        ),

    "offsets_dtype_uint64":
        (
            flow_offsets.dtype
            ==
            np.uint64
        ),

    "lengths_shape_Nx64":
        (
            packet_lengths.shape
            ==
            (
                FLOW_COUNT,
                MAX_PACKETS,
            )
        ),

    "lengths_dtype_uint16":
        (
            packet_lengths.dtype
            ==
            np.uint16
        ),

    "labels_dtype_uint8":
        (
            labels.dtype
            ==
            np.uint8
        ),

    "maximum_length_256":
        (
            int(
                packet_lengths.max()
            )
            ==
            MAX_BYTES_PER_PACKET
        ),

    "encoded_bytes_accounting":
        (
            int(
                flow_offsets[
                    -1
                ]
            )
            ==
            EXPECTED_MEMBERS[
                "Monday/encoded_bytes.bin"
            ][
                "size"
            ]
        ),
}


for name, passed in geometry_checks.items():

    print(
        f"{name:36s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    geometry_checks.values()
):

    raise RuntimeError(
        "Release geometry no longer agrees with resolved encoder semantics."
    )


# =============================================================================
# 9. FINAL SEMANTIC GATE
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-R :: FINAL SEMANTIC GATE"
)


semantic_checks = {
    "encoder_packet_rows_64":
        (
            MAX_PACKETS
            ==
            64
        ),

    "encoder_packet_width_256":
        (
            MAX_BYTES_PER_PACKET
            ==
            256
        ),

    "encoder_contains_uint8":
        (
            "uint8"
            in
            encoder_text
        ),

    "scale_for_model_float32":
        has_float32,

    "scale_for_model_div255":
        has_255_division,

    "historical_loader_unique":
        (
            HISTORICAL_LOADER_CLASS
            ==
            "Stage20CompactCorpus"
        ),

    "loader_uses_encoded_bytes":
        (
            "encoded_bytes"
            in
            loader_text
        ),

    "loader_uses_packet_lengths":
        (
            "packet_lengths"
            in
            loader_text
        ),

    "loader_uses_flow_offsets":
        (
            "flow_offsets"
            in
            loader_text
        ),

    "release_geometry_64x256":
        (
            packet_lengths.shape[
                1
            ]
            ==
            64
            and
            int(
                packet_lengths.max()
            )
            ==
            256
        ),
}


for name, passed in semantic_checks.items():

    print(
        f"{name:40s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    semantic_checks.values()
):

    failed = [
        name
        for name, passed in semantic_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Semantic gate failed:\n"
        +
        "\n".join(
            failed
        )
    )


# =============================================================================
# 10. WRITE FRESH 4C0 INVENTORY RECEIPT
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-R :: WRITE INVENTORY RECEIPT"
)


inventory = {
    "schema":
        "stage26_4c0_representation_implementation_inventory_v2_fresh_session",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4C0-RECOVERY",

    "status":
        "PASS_INVENTORY_REBUILT_AFTER_SESSION_RESET",

    "scientific_parent":
        EXPECTED_HEAD,

    "session_recovery": {
        "previous_transient_4c0_lost":
            True,

        "previous_unpushed_4c1_treated_as_lost":
            True,

        "scientific_measurement_repeated":
            False,

        "recovery_semantics":
            (
                "Authoritative existing GitHub Release corpus restored; "
                "no corpus recreated from PCAP."
            ),

        "previous_cell_completed_full_tar_sha256":
            True,

        "previous_cell_completed_all_four_member_sha256":
            True,

        "narrow_recovery_reason":
            (
                "Previous recovery stopped only because lexical source "
                "matching for encoder constants was formatting-sensitive."
            ),
    },

    "authoritative_release": {
        "tag":
            "stage20-compact-corpora-v1",

        "asset":
            "stage20-Monday-compact-corpus-v1.tar",

        "size_bytes":
            EXPECTED_TAR_SIZE,

        "sha256":
            EXPECTED_TAR_SHA256,

        "restored_from_existing_release":
            True,

        "recreated_from_pcap":
            False,

        "member_identities": {
            key: {
                "size_bytes":
                    value[
                        "size"
                    ],

                "sha256":
                    value[
                        "sha256"
                    ],
            }
            for key, value in EXPECTED_MEMBERS.items()
        },
    },

    "geometry": {
        "flow_count":
            FLOW_COUNT,

        "flow_offsets_shape":
            list(
                flow_offsets.shape
            ),

        "flow_offsets_dtype":
            str(
                flow_offsets.dtype
            ),

        "labels_shape":
            list(
                labels.shape
            ),

        "labels_dtype":
            str(
                labels.dtype
            ),

        "packet_lengths_shape":
            list(
                packet_lengths.shape
            ),

        "packet_lengths_dtype":
            str(
                packet_lengths.dtype
            ),

        "packet_slots_per_flow":
            MAX_PACKETS,

        "maximum_encoded_bytes_per_packet":
            MAX_BYTES_PER_PACKET,

        "encoded_byte_count":
            int(
                flow_offsets[
                    -1
                ]
            ),
    },

    "stage20_implementation": {
        "encoder": {
            "repo_relative_path":
                str(
                    ENCODER.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_ENCODER_SHA256,

            "resolved_packet_count_constant": {
                "name":
                    max_packets_resolution[
                        "name"
                    ],

                "value":
                    MAX_PACKETS,

                "line":
                    max_packets_resolution[
                        "line"
                    ],

                "source":
                    max_packets_resolution[
                        "source"
                    ],
            },

            "resolved_packet_width_constant": {
                "name":
                    max_bytes_resolution[
                        "name"
                    ],

                "value":
                    MAX_BYTES_PER_PACKET,

                "line":
                    max_bytes_resolution[
                        "line"
                    ],

                "source":
                    max_bytes_resolution[
                        "source"
                    ],
            },

            "scale_for_model": {
                "exists":
                    True,

                "float32_conversion":
                    True,

                "division_by_255":
                    True,

                "source":
                    scale_source,
            },
        },

        "compact_loader": {
            "repo_relative_path":
                str(
                    LOADER.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_LOADER_SHA256,

            "reconstruct_class":
                HISTORICAL_LOADER_CLASS,

            "reconstruct_args":
                loader_api[
                    "reconstruct_args"
                ],

            "reconstruct_line":
                loader_api[
                    "reconstruct_line"
                ],
        },

        "architecture_lock_sha256":
            EXPECTED_ARCH_LOCK_SHA256,

        "measurement_protocol_sha256":
            EXPECTED_MEASUREMENT_PROTOCOL_SHA256,
    },

    "resolved_representation_boundary": {
        "component":
            (
                "existing Stage20 compact GitHub Release corpus "
                "-> dense packet-image representation"
            ),

        "input_files": [
            "encoded_bytes.bin",
            "flow_offsets.npy",
            "packet_lengths.npy",
        ],

        "labels_required_for_component":
            False,

        "dense_image_shape_per_flow": [
            MAX_PACKETS,
            MAX_BYTES_PER_PACKET,
        ],

        "dense_image_dtype":
            "uint8",

        "historical_padding_mask_generated_by_loader":
            True,

        "float32_div255_part_of_this_component":
            False,

        "float32_div255_boundary":
            "MODEL_INPUT_BOUNDARY",

        "raw_packet_header_masking_part_of_this_component":
            False,

        "raw_packet_truncation_encoding_part_of_this_component":
            False,

        "complete_raw_flow_to_model_E2E_claim_allowed":
            False,

        "group_A_70_feature_extraction_profiled":
            False,
    },

    "scientific_state": {
        "pcap_accessed":
            False,

        "pcap_iterated":
            False,

        "release_asset_restored":
            True,

        "release_corpus_regenerated":
            False,

        "dense_representation_materialized":
            False,

        "representation_equivalence_test_performed":
            False,

        "representation_timing_performed":
            False,

        "models_loaded":
            False,

        "inference_performed":
            False,

        "Thursday_accessed":
            False,

        "Friday_accessed":
            False,

        "gpu_used":
            False,

        "git_modified":
            False,
    },

    "next_action":
        (
            "Re-freeze Stage26-4C1 representation protocol from durable "
            "HEAD using this fresh-session inventory SHA256, then commit "
            "and push the protocol before any Release extraction to a "
            "runtime corpus directory, equivalence execution, or timing."
        ),
}


atomic_json(
    INVENTORY_RECEIPT,
    inventory,
)


inventory_sha = sha256_file(
    INVENTORY_RECEIPT
)


print(
    "Inventory receipt:"
)

print(
    " ",
    INVENTORY_RECEIPT
)

print(
    "SHA256:"
)

print(
    " ",
    inventory_sha
)


# =============================================================================
# 11. FINAL GIT AUDIT
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-R :: FINAL GIT AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_HEAD:

    raise RuntimeError(
        "HEAD changed during 4C0 narrow recovery."
    )


if final_remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed during 4C0 narrow recovery."
    )


if final_status:

    raise RuntimeError(
        "Repository changed during 4C0 narrow recovery."
    )


# =============================================================================
# 12. CLOSURE
# =============================================================================

banner(
    "STAGE26-4C0 FRESH-SESSION RECOVERY COMPLETE"
)


print(
    "DURABLE SCIENTIFIC HEAD:"
)

print(
    " ",
    EXPECTED_HEAD
)


print(
    "\nENCODER CONSTANTS RESOLVED FROM AST:"
)

print(
    "  packet rows :",
    max_packets_resolution[
        "name"
    ],
    "=",
    MAX_PACKETS
)

print(
    "  packet bytes:",
    max_bytes_resolution[
        "name"
    ],
    "=",
    MAX_BYTES_PER_PACKET
)


print(
    "\nHISTORICAL LOADER:"
)

print(
    "  class :",
    HISTORICAL_LOADER_CLASS
)

print(
    "  method: reconstruct(self, index)"
)


print(
    "\nREPRESENTATION BOUNDARY:"
)

print(
    "  input  : existing masked Release bytes + offsets + lengths"
)

print(
    "  output : uint8 [64,256] + historical padding mask"
)

print(
    "  /255   : MODEL-INPUT BOUNDARY — NOT representation materialization"
)

print(
    "  labels : NOT REQUIRED"
)

print(
    "  raw masking/encoding : NOT INCLUDED"
)

print(
    "  Group-A 70-feature extraction : NOT PROFILED"
)

print(
    "  complete E2E claim : BLOCKED"
)


print(
    "\nAUTHORITATIVE RELEASE:"
)

print(
    "  restored from Release : YES"
)

print(
    "  recreated from PCAP   : NO"
)

print(
    "  flows                 :",
    f"{FLOW_COUNT:,}"
)

print(
    "  encoded bytes         :",
    f"{int(flow_offsets[-1]):,}"
)


print(
    "\nNEW 4C0 INVENTORY SHA256:"
)

print(
    " ",
    inventory_sha
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  REPRESENTATION PROTOCOL FROZEN : NO"
)

print(
    "  DENSE REPRESENTATION BUILT     : NO"
)

print(
    "  EQUIVALENCE EXECUTED            : NO"
)

print(
    "  REPRESENTATION TIMING           : NO"
)

print(
    "  PCAP ACCESSED                   : NO"
)

print(
    "  CORPUS REGENERATED              : NO"
)

print(
    "  MODELS LOADED                   : NO"
)

print(
    "  GPU                             : NO"
)

print(
    "  GIT MODIFIED                    : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Re-freeze Stage26-4C1 against THIS inventory SHA256"
)

print(
    "  and push the protocol immediately before any execution."
)


STAGE26-4C0-RECOVERY-R :: DURABLE STATE
Expected HEAD : 55aba2e1cc08385659479c6275234dc23b11d231
Local HEAD    : 55aba2e1cc08385659479c6275234dc23b11d231
origin/main   : 55aba2e1cc08385659479c6275234dc23b11d231
Repo clean    : True

STAGE26-4C0-RECOVERY-R :: IDENTITY GATE
Stage20 encoder                  PASS 9883fe2b27020aaff707a753123b35eb3223d21abf295d056ec233e532f94222
Stage20 compact loader           PASS a1ba15881afeb1cf4de9225a06df9ae676b95f596c8ddced7734a445ba7624d0
Stage20 architecture lock        PASS d3bba4d9d9df4432383a7f4239a8562f484cb5805b66d52794873bfaca837b3b
Stage26 measurement protocol     PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625

Monday TAR size: 595261440
Expected       : 595261440
Previously verified full SHA256:
  4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20

Full TAR / encoded_bytes hashes are NOT repeated.
They already passed in the immediately preceding recovery cell.

STAGE26-4C0-RECOVERY-R :: ENCODER AST CONST

RuntimeError: Could not structurally confirm /255 model scaling.

In [4]:
# =============================================================================
# STAGE26-4C0-RECOVERY-RR
# NARROW RECOVERY FOR NUMPY-SCALAR /255 AST FORM
#
# PREVIOUS CELLS HAVE ALREADY VERIFIED:
#   - durable HEAD 55aba2e...
#   - exact Monday GitHub Release TAR full SHA256
#   - all four Release-member SHA256 values
#   - geometry: 528,509 flows, offsets N+1, lengths [N,64]
#   - Stage20 encoder / loader / architecture hashes
#   - encoder constants ROWS=64, COLS=256
#   - historical loader class Stage20CompactCorpus
#
# LAST FAILURE ONLY:
#   AST detector expected a literal denominator:
#       ... / 255.0
#
#   Actual frozen source uses:
#       ... / np.float32(255.0)
#
# THIS CELL:
#   - confirms that exact semantic form;
#   - rechecks only small .npy Release members;
#   - writes the missing fresh-session 4C0 inventory receipt.
#
# NO:
#   - network download
#   - full TAR rehash
#   - encoded_bytes.bin rehash
#   - Release extraction to corpus directory
#   - representation materialization
#   - representation timing
#   - PCAP
#   - models
#   - GPU
#   - Git modification
# =============================================================================

from __future__ import annotations

import ast
import io
import os
import json
import hashlib
import tarfile
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np


# =============================================================================
# 0. FROZEN IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "55aba2e1cc08385659479c6275234dc23b11d231"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

REP_ROOT = (
    STAGE26_ROOT
    / "representation"
)

REP_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

INVENTORY_RECEIPT = (
    REP_ROOT
    / "stage26_4c0_representation_implementation_inventory.json"
)


MONDAY_TAR = (
    STAGE26_ROOT
    / "release_corpora"
    / "Monday"
    / "stage20-Monday-compact-corpus-v1.tar"
)

EXPECTED_TAR_SIZE = (
    595_261_440
)

EXPECTED_TAR_SHA256 = (
    "4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20"
)


EXPECTED_MEMBERS = {
    "Monday/encoded_bytes.bin": {
        "size":
            522_845_159,

        "sha256":
            "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",
    },

    "Monday/flow_offsets.npy": {
        "size":
            4_228_208,

        "sha256":
            "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",
    },

    "Monday/labels.npy": {
        "size":
            528_637,

        "sha256":
            "48792b8d6a127b35342cb0789baa6c54396f1100a60ce7225daf08d1c3530424",
    },

    "Monday/packet_lengths.npy": {
        "size":
            67_649_280,

        "sha256":
            "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
    },
}


ENCODER = (
    REPO
    / "scripts"
    / "stage20_packet_image_encoder.py"
)

EXPECTED_ENCODER_SHA256 = (
    "9883fe2b27020aaff707a753123b35eb3223d21abf295d056ec233e532f94222"
)


LOADER = (
    REPO
    / "scripts"
    / "stage20_compact_corpus.py"
)

EXPECTED_LOADER_SHA256 = (
    "a1ba15881afeb1cf4de9225a06df9ae676b95f596c8ddced7734a445ba7624d0"
)


ARCH_LOCK = (
    REPO
    / "results"
    / "stage20_1e_training"
    / "stage20_1e0_architecture_training_protocol_lock.json"
)

EXPECTED_ARCH_LOCK_SHA256 = (
    "d3bba4d9d9df4432383a7f4239a8562f484cb5805b66d52794873bfaca837b3b"
)


MEASUREMENT_PROTOCOL = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXPECTED_MEASUREMENT_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 122
    )

    print(text)

    print(
        "=" * 122
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            p.stdout
        )

    return p


def git(*args):

    return run(
        [
            "git",
            *args,
        ]
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def source_segment(
    source,
    node,
):

    value = ast.get_source_segment(
        source,
        node,
    )

    return (
        value
        if value is not None
        else "<source unavailable>"
    )


def numeric_semantic_value(node):
    """
    Resolve only deliberately-safe numeric AST forms required for provenance:
      255
      255.0
      +255.0
      -255.0
      np.float32(255.0)
      float32(255.0)
      float(255.0)
      int(255)
    """

    if isinstance(
        node,
        ast.Constant,
    ):

        if isinstance(
            node.value,
            (
                int,
                float,
            ),
        ):

            return float(
                node.value
            )


    if isinstance(
        node,
        ast.UnaryOp,
    ):

        value = numeric_semantic_value(
            node.operand
        )

        if value is None:

            return None


        if isinstance(
            node.op,
            ast.UAdd,
        ):

            return value


        if isinstance(
            node.op,
            ast.USub,
        ):

            return -value


    if isinstance(
        node,
        ast.Call,
    ):

        if (
            len(
                node.args
            )
            != 1
            or
            node.keywords
        ):

            return None


        constructor = None


        if isinstance(
            node.func,
            ast.Name,
        ):

            constructor = node.func.id


        elif isinstance(
            node.func,
            ast.Attribute,
        ):

            constructor = node.func.attr


        if constructor not in {
            "float",
            "int",
            "float16",
            "float32",
            "float64",
            "int16",
            "int32",
            "int64",
            "uint16",
            "uint32",
            "uint64",
        }:

            return None


        return numeric_semantic_value(
            node.args[
                0
            ]
        )


    return None


# =============================================================================
# 2. DURABLE STATE
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-RR :: DURABLE STATE"
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD :",
    EXPECTED_HEAD
)

print(
    "Local HEAD    :",
    head
)

print(
    "origin/main   :",
    remote
)

print(
    "Repo clean    :",
    status == ""
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected durable HEAD."
    )


if remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if INVENTORY_RECEIPT.exists():

    raise RuntimeError(
        "4C0 inventory receipt already exists; do not overwrite."
    )


# =============================================================================
# 3. EXACT IMPLEMENTATION IDENTITY
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-RR :: IMPLEMENTATION IDENTITY"
)


for label, path, expected in [
    (
        "Stage20 encoder",
        ENCODER,
        EXPECTED_ENCODER_SHA256,
    ),
    (
        "Stage20 compact loader",
        LOADER,
        EXPECTED_LOADER_SHA256,
    ),
    (
        "Stage20 architecture lock",
        ARCH_LOCK,
        EXPECTED_ARCH_LOCK_SHA256,
    ),
    (
        "Stage26 measurement protocol",
        MEASUREMENT_PROTOCOL,
        EXPECTED_MEASUREMENT_PROTOCOL_SHA256,
    ),
]:

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )

    print(
        f"{label:32s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )

    if not passed:

        raise RuntimeError(
            f"Frozen identity mismatch: {label}"
        )


if not MONDAY_TAR.exists():

    raise FileNotFoundError(
        MONDAY_TAR
    )


if int(
    MONDAY_TAR.stat().st_size
) != EXPECTED_TAR_SIZE:

    raise RuntimeError(
        "Monday TAR size changed."
    )


print(
    "\nMonday TAR size       : PASS"
)

print(
    "Full TAR SHA256       : already verified in previous recovery cell"
)

print(
    "encoded_bytes SHA256  : already verified in previous recovery cell"
)

print(
    "Network download      : NO"
)


# =============================================================================
# 4. RESOLVE ROWS / COLS FROM EXACT AST
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-RR :: ENCODER GEOMETRY AST"
)


encoder_text = ENCODER.read_text(
    encoding="utf-8"
)

encoder_tree = ast.parse(
    encoder_text
)


resolved_constants = {}


for node in encoder_tree.body:

    target_name = None


    if isinstance(
        node,
        ast.Assign,
    ):

        if (
            len(
                node.targets
            )
            ==
            1
            and
            isinstance(
                node.targets[
                    0
                ],
                ast.Name,
            )
        ):

            target_name = node.targets[
                0
            ].id


    elif isinstance(
        node,
        ast.AnnAssign,
    ):

        if isinstance(
            node.target,
            ast.Name,
        ):

            target_name = node.target.id


    if target_name not in {
        "ROWS",
        "COLS",
    }:

        continue


    value = ast.literal_eval(
        node.value
    )


    resolved_constants[
        target_name
    ] = {
        "value":
            int(
                value
            ),

        "line":
            int(
                node.lineno
            ),

        "source":
            source_segment(
                encoder_text,
                node,
            ),
    }


print(
    "ROWS:",
    resolved_constants.get(
        "ROWS"
    )
)

print(
    "COLS:",
    resolved_constants.get(
        "COLS"
    )
)


if resolved_constants.get(
    "ROWS",
    {}
).get(
    "value"
) != 64:

    raise RuntimeError(
        "Frozen ROWS is not 64."
    )


if resolved_constants.get(
    "COLS",
    {}
).get(
    "value"
) != 256:

    raise RuntimeError(
        "Frozen COLS is not 256."
    )


ROWS = (
    64
)

COLS = (
    256
)


# =============================================================================
# 5. EXACT scale_for_model SEMANTIC AUDIT
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-RR :: /255 SEMANTIC AST AUDIT"
)


scale_nodes = [
    node
    for node in encoder_tree.body
    if (
        isinstance(
            node,
            ast.FunctionDef,
        )
        and
        node.name
        ==
        "scale_for_model"
    )
]


if len(
    scale_nodes
) != 1:

    raise RuntimeError(
        "Expected exactly one scale_for_model()."
    )


scale_node = scale_nodes[
    0
]

scale_source = source_segment(
    encoder_text,
    scale_node,
)


print(
    scale_source
)


float32_evidence = []

division_evidence = []


for node in ast.walk(
    scale_node
):

    # -------------------------------------------------------------------------
    # float32 evidence
    # -------------------------------------------------------------------------

    if (
        isinstance(
            node,
            ast.Attribute,
        )
        and
        node.attr
        ==
        "float32"
    ):

        float32_evidence.append(
            source_segment(
                encoder_text,
                node,
            )
        )


    if (
        isinstance(
            node,
            ast.Name,
        )
        and
        node.id
        ==
        "float32"
    ):

        float32_evidence.append(
            source_segment(
                encoder_text,
                node,
            )
        )


    # -------------------------------------------------------------------------
    # Division denominator semantic resolution
    # -------------------------------------------------------------------------

    if (
        isinstance(
            node,
            ast.BinOp,
        )
        and
        isinstance(
            node.op,
            ast.Div,
        )
    ):

        denominator_value = numeric_semantic_value(
            node.right
        )


        division_evidence.append(
            {
                "expression":
                    source_segment(
                        encoder_text,
                        node,
                    ),

                "denominator_source":
                    source_segment(
                        encoder_text,
                        node.right,
                    ),

                "denominator_semantic_value":
                    denominator_value,
            }
        )


print(
    "\nfloat32 AST evidence:"
)


for item in float32_evidence:

    print(
        " ",
        item
    )


print(
    "\nDivision AST evidence:"
)


for item in division_evidence:

    print(
        " ",
        item
    )


has_float32 = (
    len(
        float32_evidence
    )
    >
    0
)


division_by_255_matches = [
    item
    for item in division_evidence
    if (
        item[
            "denominator_semantic_value"
        ]
        is not None
        and
        float(
            item[
                "denominator_semantic_value"
            ]
        )
        ==
        255.0
    )
]


has_semantic_div255 = (
    len(
        division_by_255_matches
    )
    >=
    1
)


print(
    "\nfloat32 conversion confirmed :",
    has_float32
)

print(
    "semantic division by 255    :",
    has_semantic_div255
)


if not has_float32:

    raise RuntimeError(
        "float32 conversion not confirmed."
    )


if not has_semantic_div255:

    raise RuntimeError(
        "Semantic /255 operation not confirmed."
    )


# Require the exact observed denominator form to be preserved in provenance.
denominator_sources = {
    item[
        "denominator_source"
    ]
    for item in division_by_255_matches
}


print(
    "Resolved /255 denominator(s):"
)


for item in sorted(
    denominator_sources
):

    print(
        " ",
        item
    )


if "np.float32(255.0)" not in denominator_sources:

    raise RuntimeError(
        "Expected frozen denominator np.float32(255.0) not found."
    )


print(
    "\n[PASS] Exact model scaling is float32 / np.float32(255.0)."
)


# =============================================================================
# 6. HISTORICAL LOADER API
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-RR :: HISTORICAL LOADER API"
)


loader_text = LOADER.read_text(
    encoding="utf-8"
)

loader_tree = ast.parse(
    loader_text
)


loader_candidates = []


for node in loader_tree.body:

    if not isinstance(
        node,
        ast.ClassDef,
    ):

        continue


    for child in node.body:

        if (
            isinstance(
                child,
                ast.FunctionDef,
            )
            and
            child.name
            ==
            "reconstruct"
        ):

            loader_candidates.append(
                {
                    "class_name":
                        node.name,

                    "class_line":
                        int(
                            node.lineno
                        ),

                    "reconstruct_line":
                        int(
                            child.lineno
                        ),

                    "reconstruct_args":
                        [
                            arg.arg
                            for arg in child.args.args
                        ],
                }
            )


print(
    "Candidates:",
    loader_candidates
)


if len(
    loader_candidates
) != 1:

    raise RuntimeError(
        "Historical reconstruct() API is not unique."
    )


loader_api = loader_candidates[
    0
]

HISTORICAL_LOADER_CLASS = loader_api[
    "class_name"
]


if HISTORICAL_LOADER_CLASS != "Stage20CompactCorpus":

    raise RuntimeError(
        "Unexpected historical compact-loader class."
    )


if loader_api[
    "reconstruct_args"
][:2] != [
    "self",
    "index",
]:

    raise RuntimeError(
        "Unexpected reconstruct() signature."
    )


print(
    "\nHistorical loader:",
    HISTORICAL_LOADER_CLASS
)

print(
    "Method           : reconstruct(self, index)"
)

print(
    "[PASS]"
)


# =============================================================================
# 7. SMALL MEMBER RECHECK ONLY
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-RR :: SMALL RELEASE MEMBER GATE"
)


small_member_names = [
    "Monday/flow_offsets.npy",
    "Monday/labels.npy",
    "Monday/packet_lengths.npy",
]


payloads = {}


with tarfile.open(
    MONDAY_TAR,
    mode="r:",
) as tf:

    members = {
        member.name:
            member
        for member in tf.getmembers()
        if member.isfile()
    }


    if set(
        members
    ) != set(
        EXPECTED_MEMBERS
    ):

        raise RuntimeError(
            "Monday Release member set changed."
        )


    for member_name in small_member_names:

        f = tf.extractfile(
            members[
                member_name
            ]
        )


        if f is None:

            raise RuntimeError(
                f"Unable to read {member_name}"
            )


        raw = f.read()

        digest = sha256_bytes(
            raw
        )

        frozen = EXPECTED_MEMBERS[
            member_name
        ]


        passed = (
            len(
                raw
            )
            ==
            frozen[
                "size"
            ]
            and
            digest
            ==
            frozen[
                "sha256"
            ]
        )


        print(
            f"{member_name:34s} "
            f"{'PASS' if passed else 'FAIL'} "
            f"{len(raw):12,d} B "
            f"{digest}"
        )


        if not passed:

            raise RuntimeError(
                f"Small Release member changed: {member_name}"
            )


        payloads[
            member_name
        ] = raw


flow_offsets = np.load(
    io.BytesIO(
        payloads[
            "Monday/flow_offsets.npy"
        ]
    ),
    allow_pickle=False,
)

labels = np.load(
    io.BytesIO(
        payloads[
            "Monday/labels.npy"
        ]
    ),
    allow_pickle=False,
)

packet_lengths = np.load(
    io.BytesIO(
        payloads[
            "Monday/packet_lengths.npy"
        ]
    ),
    allow_pickle=False,
)


FLOW_COUNT = int(
    labels.size
)


geometry_checks = {
    "flow_count_528509":
        (
            FLOW_COUNT
            ==
            528_509
        ),

    "offsets_N_plus_1":
        (
            flow_offsets.shape
            ==
            (
                FLOW_COUNT + 1,
            )
        ),

    "offsets_uint64":
        (
            flow_offsets.dtype
            ==
            np.uint64
        ),

    "lengths_Nx64":
        (
            packet_lengths.shape
            ==
            (
                FLOW_COUNT,
                ROWS,
            )
        ),

    "lengths_uint16":
        (
            packet_lengths.dtype
            ==
            np.uint16
        ),

    "labels_uint8":
        (
            labels.dtype
            ==
            np.uint8
        ),

    "max_length_256":
        (
            int(
                packet_lengths.max()
            )
            ==
            COLS
        ),

    "encoded_byte_accounting":
        (
            int(
                flow_offsets[
                    -1
                ]
            )
            ==
            EXPECTED_MEMBERS[
                "Monday/encoded_bytes.bin"
            ][
                "size"
            ]
        ),
}


for name, passed in geometry_checks.items():

    print(
        f"{name:34s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    geometry_checks.values()
):

    raise RuntimeError(
        "Release geometry gate failed."
    )


# =============================================================================
# 8. FINAL BOUNDARY RESOLUTION
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-RR :: RESOLVED COMPONENT BOUNDARY"
)


print(
    "Authoritative input:"
)

print(
    "  existing GitHub Release compact corpus"
)

print(
    "  encoded_bytes.bin + flow_offsets.npy + packet_lengths.npy"
)


print(
    "\nRepresentation materialization:"
)

print(
    "  output image geometry : uint8 [64,256]"
)

print(
    "  padding information   : historical compact loader reconstruction"
)


print(
    "\nModel-boundary scaling:"
)

print(
    "  float32 conversion    : YES"
)

print(
    "  denominator           : np.float32(255.0)"
)

print(
    "  INCLUDED IN 4C representation timing: NO"
)


print(
    "\nNot represented by this compact-corpus component:"
)

print(
    "  raw PCAP parsing")
print(
    "  flow reconstruction")
print(
    "  packet selection")
print(
    "  header masking")
print(
    "  truncation/encoding into encoded_bytes.bin")
print(
    "  Group-A 70-feature extraction")
print(
    "  model inference")


# =============================================================================
# 9. WRITE FRESH-SESSION 4C0 INVENTORY
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-RR :: WRITE INVENTORY"
)


inventory = {
    "schema":
        "stage26_4c0_representation_implementation_inventory_v2_fresh_session",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4C0-RECOVERY",

    "status":
        "PASS_INVENTORY_REBUILT_AFTER_SESSION_RESET",

    "scientific_parent":
        EXPECTED_HEAD,

    "session_recovery": {
        "previous_transient_4c0_lost":
            True,

        "previous_unpushed_4c1_treated_as_lost":
            True,

        "scientific_measurement_repeated":
            False,

        "release_redownloaded_in_this_RR_cell":
            False,

        "full_tar_rehashed_in_this_RR_cell":
            False,

        "encoded_bytes_rehashed_in_this_RR_cell":
            False,

        "previous_recovery_full_tar_verification_passed":
            True,

        "previous_recovery_all_four_member_verification_passed":
            True,

        "narrow_recovery_reason":
            (
                "Prior AST detector did not recognize np.float32(255.0) "
                "as a numeric denominator equal to 255."
            ),
    },

    "authoritative_release": {
        "tag":
            "stage20-compact-corpora-v1",

        "asset":
            "stage20-Monday-compact-corpus-v1.tar",

        "size_bytes":
            EXPECTED_TAR_SIZE,

        "sha256":
            EXPECTED_TAR_SHA256,

        "restored_from_existing_GitHub_release":
            True,

        "recreated_from_pcap":
            False,

        "member_identities": {
            key: {
                "size_bytes":
                    value[
                        "size"
                    ],

                "sha256":
                    value[
                        "sha256"
                    ],
            }
            for key, value in EXPECTED_MEMBERS.items()
        },
    },

    "geometry": {
        "flow_count":
            FLOW_COUNT,

        "flow_offsets_shape":
            list(
                flow_offsets.shape
            ),

        "flow_offsets_dtype":
            str(
                flow_offsets.dtype
            ),

        "labels_shape":
            list(
                labels.shape
            ),

        "labels_dtype":
            str(
                labels.dtype
            ),

        "packet_lengths_shape":
            list(
                packet_lengths.shape
            ),

        "packet_lengths_dtype":
            str(
                packet_lengths.dtype
            ),

        "packet_slots_per_flow":
            ROWS,

        "maximum_encoded_bytes_per_packet":
            COLS,

        "encoded_byte_count":
            int(
                flow_offsets[
                    -1
                ]
            ),
    },

    "stage20_implementation": {
        "encoder": {
            "repo_relative_path":
                str(
                    ENCODER.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_ENCODER_SHA256,

            "ROWS": {
                "value":
                    ROWS,

                "line":
                    resolved_constants[
                        "ROWS"
                    ][
                        "line"
                    ],

                "source":
                    resolved_constants[
                        "ROWS"
                    ][
                        "source"
                    ],
            },

            "COLS": {
                "value":
                    COLS,

                "line":
                    resolved_constants[
                        "COLS"
                    ][
                        "line"
                    ],

                "source":
                    resolved_constants[
                        "COLS"
                    ][
                        "source"
                    ],
            },

            "scale_for_model": {
                "exists":
                    True,

                "float32_conversion":
                    True,

                "division_by_255":
                    True,

                "exact_denominator_source":
                    "np.float32(255.0)",

                "semantic_denominator_value":
                    255.0,

                "source":
                    scale_source,
            },
        },

        "compact_loader": {
            "repo_relative_path":
                str(
                    LOADER.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_LOADER_SHA256,

            "reconstruct_class":
                HISTORICAL_LOADER_CLASS,

            "reconstruct_args":
                loader_api[
                    "reconstruct_args"
                ],

            "reconstruct_line":
                loader_api[
                    "reconstruct_line"
                ],
        },

        "architecture_lock_sha256":
            EXPECTED_ARCH_LOCK_SHA256,

        "measurement_protocol_sha256":
            EXPECTED_MEASUREMENT_PROTOCOL_SHA256,
    },

    "resolved_representation_boundary": {
        "component":
            (
                "existing Stage20 compact GitHub Release corpus "
                "-> dense packet-image materialization"
            ),

        "input_files": [
            "encoded_bytes.bin",
            "flow_offsets.npy",
            "packet_lengths.npy",
        ],

        "labels_required_for_component":
            False,

        "dense_image_shape_per_flow": [
            ROWS,
            COLS,
        ],

        "dense_image_dtype":
            "uint8",

        "historical_padding_mask_generated_by_loader":
            True,

        "float32_div255_part_of_this_component":
            False,

        "float32_div255_boundary":
            "MODEL_INPUT_BOUNDARY",

        "raw_packet_header_masking_part_of_this_component":
            False,

        "raw_packet_truncation_encoding_part_of_this_component":
            False,

        "group_A_70_feature_extraction_profiled":
            False,

        "complete_raw_flow_to_model_E2E_claim_allowed":
            False,
    },

    "scientific_state": {
        "pcap_accessed":
            False,

        "pcap_iterated":
            False,

        "release_asset_restored":
            True,

        "release_corpus_regenerated":
            False,

        "dense_representation_materialized":
            False,

        "representation_equivalence_test_performed":
            False,

        "representation_timing_performed":
            False,

        "models_loaded":
            False,

        "inference_performed":
            False,

        "Thursday_accessed":
            False,

        "Friday_accessed":
            False,

        "gpu_used":
            False,

        "git_modified":
            False,
    },

    "next_action":
        (
            "Freeze Stage26-4C1 representation protocol using this exact "
            "fresh-session inventory SHA256, then commit and push the lock "
            "before any equivalence execution or representation timing."
        ),
}


atomic_json(
    INVENTORY_RECEIPT,
    inventory,
)


inventory_sha = sha256_file(
    INVENTORY_RECEIPT
)


print(
    "Inventory:"
)

print(
    " ",
    INVENTORY_RECEIPT
)

print(
    "SHA256:"
)

print(
    " ",
    inventory_sha
)


# =============================================================================
# 10. FINAL GIT AUDIT
# =============================================================================

banner(
    "STAGE26-4C0-RECOVERY-RR :: FINAL GIT AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_HEAD:

    raise RuntimeError(
        "HEAD changed."
    )


if final_remote != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main changed."
    )


if final_status:

    raise RuntimeError(
        "Repository changed."
    )


# =============================================================================
# 11. CLOSURE
# =============================================================================

banner(
    "STAGE26-4C0 FRESH-SESSION RECOVERY COMPLETE"
)


print(
    "DURABLE SCIENTIFIC HEAD:"
)

print(
    " ",
    EXPECTED_HEAD
)


print(
    "\nFROZEN STAGE20 GEOMETRY:"
)

print(
    "  ROWS =",
    ROWS
)

print(
    "  COLS =",
    COLS
)

print(
    "  flows =",
    f"{FLOW_COUNT:,}"
)

print(
    "  encoded bytes =",
    f"{int(flow_offsets[-1]):,}"
)


print(
    "\nMODEL-SCALING BOUNDARY:"
)

print(
    "  conversion  : np.float32"
)

print(
    "  denominator : np.float32(255.0)"
)

print(
    "  semantic /255 confirmed : YES"
)

print(
    "  included in Stage26-4C materialization: NO"
)


print(
    "\nHISTORICAL LOADER:"
)

print(
    "  class :",
    HISTORICAL_LOADER_CLASS
)

print(
    "  API   : reconstruct(self, index)"
)


print(
    "\nAUTHORITATIVE CORPUS:"
)

print(
    "  GitHub Release restored : YES"
)

print(
    "  recreated from PCAP     : NO"
)

print(
    "  downloaded in this cell : NO"
)


print(
    "\nNEW 4C0 INVENTORY SHA256:"
)

print(
    " ",
    inventory_sha
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  4C0 INVENTORY COMPLETE          : YES"
)

print(
    "  REPRESENTATION PROTOCOL FROZEN : NO"
)

print(
    "  REPRESENTATION MATERIALIZED    : NO"
)

print(
    "  EQUIVALENCE EXECUTED           : NO"
)

print(
    "  REPRESENTATION TIMING          : NO"
)

print(
    "  PCAP ACCESSED                  : NO"
)

print(
    "  CORPUS REGENERATED             : NO"
)

print(
    "  MODELS LOADED                  : NO"
)

print(
    "  GPU                            : NO"
)

print(
    "  GIT MODIFIED                   : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Re-freeze Stage26-4C1 against the NEW inventory SHA256"
)

print(
    "  and immediately Git-anchor it before any execution."
)


STAGE26-4C0-RECOVERY-RR :: DURABLE STATE
Expected HEAD : 55aba2e1cc08385659479c6275234dc23b11d231
Local HEAD    : 55aba2e1cc08385659479c6275234dc23b11d231
origin/main   : 55aba2e1cc08385659479c6275234dc23b11d231
Repo clean    : True

STAGE26-4C0-RECOVERY-RR :: IMPLEMENTATION IDENTITY
Stage20 encoder                  PASS 9883fe2b27020aaff707a753123b35eb3223d21abf295d056ec233e532f94222
Stage20 compact loader           PASS a1ba15881afeb1cf4de9225a06df9ae676b95f596c8ddced7734a445ba7624d0
Stage20 architecture lock        PASS d3bba4d9d9df4432383a7f4239a8562f484cb5805b66d52794873bfaca837b3b
Stage26 measurement protocol     PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625

Monday TAR size       : PASS
Full TAR SHA256       : already verified in previous recovery cell
encoded_bytes SHA256  : already verified in previous recovery cell
Network download      : NO

STAGE26-4C0-RECOVERY-RR :: ENCODER GEOMETRY AST
ROWS: {'value': 64, 'line': 20, 'source': 'ROWS = 64'}
COLS: {

In [5]:
# =============================================================================
# STAGE26-4C1-FREEZE-GIT
# RE-FREEZE + IMMEDIATELY REMOTELY ANCHOR REPRESENTATION PROTOCOL
#
# DURABLE SCIENTIFIC PARENT:
#   55aba2e1cc08385659479c6275234dc23b11d231
#
# FRESH RECOVERY INVENTORY:
#   b32a2a21407261d0c3628e80d1c39aab1b6921cf144b9ce5daea28392ec989a5
#
# IMPORTANT:
#   This cell derives the historical Stage20CompactCorpus.reconstruct()
#   output geometry directly from the exact hashed loader source.
#
# THIS CELL:
#   - verifies scientific parent + fresh 4C0 inventory;
#   - derives historical image/mask contract from AST;
#   - freezes the representation worker and protocol;
#   - copies the transient 4C0 inventory into the durable lock package;
#   - creates manifest;
#   - commits;
#   - pushes;
#   - remotely byte-verifies every lock artifact.
#
# ABSOLUTE RULES:
#   - NO Release TAR extraction.
#   - NO representation materialization.
#   - NO equivalence execution.
#   - NO representation timing.
#   - NO PCAP access.
#   - NO corpus regeneration.
#   - NO labels.
#   - NO models.
#   - NO inference.
#   - NO Thursday / Friday.
#   - NO GPU.
#
# FIRST REPRESENTATION TIMING REMAINS FORBIDDEN.
# =============================================================================

from __future__ import annotations

import ast
import os
import json
import stat
import shutil
import hashlib
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. FROZEN IDENTITIES / PATHS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

REP_RUNTIME_ROOT = (
    STAGE26_ROOT
    / "representation"
)

EXPECTED_PARENT = (
    "55aba2e1cc08385659479c6275234dc23b11d231"
)

COMMIT_SUBJECT = (
    "stage26: freeze representation profiling protocol"
)


# -----------------------------------------------------------------------------
# Fresh-session Stage26-4C0 inventory
# -----------------------------------------------------------------------------

INVENTORY_SOURCE = (
    REP_RUNTIME_ROOT
    / "stage26_4c0_representation_implementation_inventory.json"
)

EXPECTED_INVENTORY_SHA256 = (
    "b32a2a21407261d0c3628e80d1c39aab1b6921cf144b9ce5daea28392ec989a5"
)


# -----------------------------------------------------------------------------
# Durable historical implementation
# -----------------------------------------------------------------------------

ENCODER = (
    REPO
    / "scripts"
    / "stage20_packet_image_encoder.py"
)

EXPECTED_ENCODER_SHA256 = (
    "9883fe2b27020aaff707a753123b35eb3223d21abf295d056ec233e532f94222"
)

LOADER = (
    REPO
    / "scripts"
    / "stage20_compact_corpus.py"
)

EXPECTED_LOADER_SHA256 = (
    "a1ba15881afeb1cf4de9225a06df9ae676b95f596c8ddced7734a445ba7624d0"
)

ARCH_LOCK = (
    REPO
    / "results"
    / "stage20_1e_training"
    / "stage20_1e0_architecture_training_protocol_lock.json"
)

EXPECTED_ARCH_LOCK_SHA256 = (
    "d3bba4d9d9df4432383a7f4239a8562f484cb5805b66d52794873bfaca837b3b"
)

MEASUREMENT_PROTOCOL = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXPECTED_MEASUREMENT_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)


# -----------------------------------------------------------------------------
# Durable Stage26-4B3 closure
# -----------------------------------------------------------------------------

RAW_TIMING_MANIFEST = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4b3_cpu1_extraction_timing"
    / "stage26_4b3_cpu1_extraction_manifest.json"
)


# -----------------------------------------------------------------------------
# Existing authoritative Release asset — presence only in this cell.
# -----------------------------------------------------------------------------

MONDAY_TAR = (
    STAGE26_ROOT
    / "release_corpora"
    / "Monday"
    / "stage20-Monday-compact-corpus-v1.tar"
)

EXPECTED_TAR_BYTES = (
    595_261_440
)

EXPECTED_TAR_SHA256 = (
    "4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20"
)

RELEASE_TAG = (
    "stage20-compact-corpora-v1"
)

RELEASE_ASSET = (
    "stage20-Monday-compact-corpus-v1.tar"
)


MEMBER_IDENTITIES = {
    "encoded_bytes.bin": {
        "tar_member":
            "Monday/encoded_bytes.bin",

        "size_bytes":
            522_845_159,

        "sha256":
            "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",
    },

    "flow_offsets.npy": {
        "tar_member":
            "Monday/flow_offsets.npy",

        "size_bytes":
            4_228_208,

        "sha256":
            "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",
    },

    "labels.npy": {
        "tar_member":
            "Monday/labels.npy",

        "size_bytes":
            528_637,

        "sha256":
            "48792b8d6a127b35342cb0789baa6c54396f1100a60ce7225daf08d1c3530424",
    },

    "packet_lengths.npy": {
        "tar_member":
            "Monday/packet_lengths.npy",

        "size_bytes":
            67_649_280,

        "sha256":
            "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
    },
}


FLOW_COUNT = (
    528_509
)

ROWS = (
    64
)

COLS = (
    256
)

ENCODED_BYTE_COUNT = (
    522_845_159
)


# =============================================================================
# 1. FROZEN STAGE26-4C EXECUTION POLICY
# =============================================================================

SEED = (
    26_042
)

BATCH_SIZES = [
    1,
    64,
    256,
    1024,
    8192,
]

MAX_BATCH_SIZE = max(
    BATCH_SIZES
)

SAMPLE_START = (
    SEED
    %
    (
        FLOW_COUNT
        -
        MAX_BATCH_SIZE
        +
        1
    )
)

SAMPLE_END_EXCLUSIVE = (
    SAMPLE_START
    +
    MAX_BATCH_SIZE
)


if SAMPLE_START != 26_042:

    raise RuntimeError(
        "Unexpected deterministic Stage26 sample start."
    )


ITERATION_POLICY = {
    1: {
        "warmup_runs":
            50,

        "timed_runs":
            200,

        "role":
            "LATENCY_PRIMARY",
    },

    64: {
        "warmup_runs":
            30,

        "timed_runs":
            150,

        "role":
            "MICROBATCH",
    },

    256: {
        "warmup_runs":
            20,

        "timed_runs":
            100,

        "role":
            "STANDARD_BATCH",
    },

    1024: {
        "warmup_runs":
            10,

        "timed_runs":
            50,

        "role":
            "LARGE_BATCH",
    },

    8192: {
        "warmup_runs":
            5,

        "timed_runs":
            20,

        "role":
            "CAPACITY_BATCH",
    },
}


CPU_AFFINITY = [
    0
]

THREAD_COUNT = (
    1
)

CONDITION_TIMEOUT_SECONDS = (
    600
)

EQUIVALENCE_FLOW_COUNT = (
    128
)

EQUIVALENCE_START = (
    SAMPLE_START
)


ENVIRONMENT_GATE = {
    "cpu_utilization_percent_max":
        20.0,

    "available_ram_gib_min":
        8.0,

    "cpu_utilization_sampling_seconds":
        1.0,

    "retry_count":
        3,

    "retry_cooldown_seconds":
        5,

    "if_gate_never_passes":
        "INVALID_ENVIRONMENT",
}


# =============================================================================
# 2. DURABLE LOCK PACKAGE
# =============================================================================

LOCK_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_4c_representation_protocol_lock"
)

LOCK_DIR = (
    REPO
    / LOCK_REL
)

INVENTORY_COPY = (
    LOCK_DIR
    / "stage26_4c0_representation_implementation_inventory.json"
)

WORKER_PATH = (
    LOCK_DIR
    / "stage26_representation_worker_v1.py"
)

PROTOCOL_PATH = (
    LOCK_DIR
    / "stage26_representation_protocol.json"
)

BOUNDARY_PATH = (
    LOCK_DIR
    / "stage26_representation_boundary_map.json"
)

PROVENANCE_PATH = (
    LOCK_DIR
    / "stage26_4c_upstream_provenance.json"
)

FREEZE_RECEIPT_PATH = (
    LOCK_DIR
    / "stage26_4c1_representation_protocol_freeze_receipt.json"
)

MANIFEST_PATH = (
    LOCK_DIR
    / "stage26_4c_representation_lock_manifest.json"
)


# =============================================================================
# 3. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 122
    )

    print(text)

    print(
        "=" * 122
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        check=True,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_text(
    path,
    text,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        f.write(
            text
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def atomic_json(
    path,
    obj,
):

    atomic_text(
        path,
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
        )
        +
        "\n",
    )


def source_segment(
    source,
    node,
):

    value = ast.get_source_segment(
        source,
        node,
    )

    return (
        value
        if value is not None
        else "<source unavailable>"
    )


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        check=True,
        text=False,
    )

    return bytes(
        p.stdout
    )


def call_name(node):

    if isinstance(
        node,
        ast.Name,
    ):

        return node.id


    if isinstance(
        node,
        ast.Attribute,
    ):

        left = call_name(
            node.value
        )

        if left:

            return (
                left
                +
                "."
                +
                node.attr
            )

        return node.attr


    return None


def shape_from_ast(node):

    # ROWS
    if isinstance(
        node,
        ast.Name,
    ):

        if node.id == "ROWS":

            return [
                64
            ]

        if node.id == "COLS":

            return [
                256
            ]


    # literal integer
    if isinstance(
        node,
        ast.Constant,
    ):

        if isinstance(
            node.value,
            int,
        ):

            return [
                int(
                    node.value
                )
            ]


    # (ROWS, COLS), (ROWS,), etc.
    if isinstance(
        node,
        (
            ast.Tuple,
            ast.List,
        ),
    ):

        result = []


        for element in node.elts:

            if isinstance(
                element,
                ast.Name,
            ):

                if element.id == "ROWS":

                    result.append(
                        64
                    )

                elif element.id == "COLS":

                    result.append(
                        256
                    )

                else:

                    return None


            elif isinstance(
                element,
                ast.Constant,
            ) and isinstance(
                element.value,
                int,
            ):

                result.append(
                    int(
                        element.value
                    )
                )


            else:

                return None


        return result


    return None


def dtype_from_call(node):

    if not isinstance(
        node,
        ast.Call,
    ):

        return None


    for kw in node.keywords:

        if kw.arg != "dtype":

            continue


        name = call_name(
            kw.value
        )


        if name is not None:

            return name


        if isinstance(
            kw.value,
            ast.Constant,
        ):

            return str(
                kw.value.value
            )


    return None


# =============================================================================
# 4. SCIENTIFIC PARENT GATE
# =============================================================================

banner(
    "STAGE26-4C1-FREEZE-GIT :: SCIENTIFIC PARENT"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected scientific parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before 4C1 freeze."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if LOCK_DIR.exists():

    raise RuntimeError(
        "Stage26-4C representation lock directory already exists. "
        "Do not overwrite."
    )


# =============================================================================
# 5. FRESH 4C0 INVENTORY GATE
# =============================================================================

banner(
    "STAGE26-4C1-FREEZE-GIT :: FRESH 4C0 INVENTORY"
)


if not INVENTORY_SOURCE.exists():

    raise FileNotFoundError(
        INVENTORY_SOURCE
    )


inventory_sha = sha256_file(
    INVENTORY_SOURCE
)


print(
    "Expected:",
    EXPECTED_INVENTORY_SHA256
)

print(
    "Actual  :",
    inventory_sha
)


if inventory_sha != EXPECTED_INVENTORY_SHA256:

    raise RuntimeError(
        "Fresh 4C0 inventory identity changed."
    )


inventory = json.loads(
    INVENTORY_SOURCE.read_text(
        encoding="utf-8"
    )
)


inventory_checks = {
    "status_PASS":
        (
            inventory[
                "status"
            ]
            ==
            "PASS_INVENTORY_REBUILT_AFTER_SESSION_RESET"
        ),

    "scientific_parent":
        (
            inventory[
                "scientific_parent"
            ]
            ==
            EXPECTED_PARENT
        ),

    "release_restored":
        (
            inventory[
                "scientific_state"
            ][
                "release_asset_restored"
            ]
            is True
        ),

    "corpus_not_regenerated":
        (
            inventory[
                "scientific_state"
            ][
                "release_corpus_regenerated"
            ]
            is False
        ),

    "representation_not_materialized":
        (
            inventory[
                "scientific_state"
            ][
                "dense_representation_materialized"
            ]
            is False
        ),

    "equivalence_not_run":
        (
            inventory[
                "scientific_state"
            ][
                "representation_equivalence_test_performed"
            ]
            is False
        ),

    "timing_not_run":
        (
            inventory[
                "scientific_state"
            ][
                "representation_timing_performed"
            ]
            is False
        ),

    "pcap_not_accessed":
        (
            inventory[
                "scientific_state"
            ][
                "pcap_accessed"
            ]
            is False
        ),

    "gpu_not_used":
        (
            inventory[
                "scientific_state"
            ][
                "gpu_used"
            ]
            is False
        ),

    "release_sha":
        (
            inventory[
                "authoritative_release"
            ][
                "sha256"
            ]
            ==
            EXPECTED_TAR_SHA256
        ),
}


for name, passed in inventory_checks.items():

    print(
        f"{name:40s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    inventory_checks.values()
):

    raise RuntimeError(
        "Fresh 4C0 inventory scientific gate failed."
    )


# =============================================================================
# 6. DURABLE IMPLEMENTATION + RAW EXTRACTION CLOSURE
# =============================================================================

banner(
    "STAGE26-4C1-FREEZE-GIT :: UPSTREAM PROVENANCE"
)


for label, path, expected in [
    (
        "Stage20 encoder",
        ENCODER,
        EXPECTED_ENCODER_SHA256,
    ),
    (
        "Stage20 compact loader",
        LOADER,
        EXPECTED_LOADER_SHA256,
    ),
    (
        "Stage20 architecture lock",
        ARCH_LOCK,
        EXPECTED_ARCH_LOCK_SHA256,
    ),
    (
        "Stage26 measurement protocol",
        MEASUREMENT_PROTOCOL,
        EXPECTED_MEASUREMENT_PROTOCOL_SHA256,
    ),
]:

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:32s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Upstream identity failed: {label}"
        )


if not RAW_TIMING_MANIFEST.exists():

    raise FileNotFoundError(
        RAW_TIMING_MANIFEST
    )


raw_manifest_sha = sha256_file(
    RAW_TIMING_MANIFEST
)

raw_manifest = json.loads(
    RAW_TIMING_MANIFEST.read_text(
        encoding="utf-8"
    )
)


raw_checks = {
    "raw_measurement_pass":
        (
            raw_manifest[
                "status"
            ]
            ==
            "PASS_FIVE_FROZEN_REPETITIONS_READY_FOR_GIT_ANCHOR"
            or
            raw_manifest[
                "status"
            ]
            ==
            "PASS_FIVE_FROZEN_REPETITIONS"
        ),

    "raw_extraction_measured":
        (
            raw_manifest[
                "scientific_boundaries"
            ][
                "raw_extraction_performance_measured"
            ]
            is True
        ),

    "raw_corpus_not_regenerated":
        (
            raw_manifest[
                "scientific_boundaries"
            ][
                "release_corpus_regenerated"
            ]
            is False
        ),
}


for name, passed in raw_checks.items():

    print(
        f"{name:40s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    raw_checks.values()
):

    raise RuntimeError(
        "Stage26-4B3 closure gate failed."
    )


# Release presence only — no rehash/extraction.
if not MONDAY_TAR.exists():

    raise FileNotFoundError(
        MONDAY_TAR
    )


if int(
    MONDAY_TAR.stat().st_size
) != EXPECTED_TAR_BYTES:

    raise RuntimeError(
        "Restored Monday Release TAR size changed."
    )


print(
    "\nRelease TAR full SHA already verified by 4C0:"
)

print(
    " ",
    EXPECTED_TAR_SHA256
)

print(
    "Release TAR extraction in this cell: NO"
)


# =============================================================================
# 7. DERIVE HISTORICAL RECONSTRUCT CONTRACT FROM EXACT SOURCE
# =============================================================================

banner(
    "STAGE26-4C1-FREEZE-GIT :: HISTORICAL RECONSTRUCT CONTRACT"
)


loader_text = LOADER.read_text(
    encoding="utf-8"
)

loader_tree = ast.parse(
    loader_text
)


reconstruct_candidates = []


for node in loader_tree.body:

    if not isinstance(
        node,
        ast.ClassDef,
    ):

        continue


    for child in node.body:

        if (
            isinstance(
                child,
                ast.FunctionDef,
            )
            and
            child.name
            ==
            "reconstruct"
        ):

            reconstruct_candidates.append(
                (
                    node,
                    child,
                )
            )


if len(
    reconstruct_candidates
) != 1:

    raise RuntimeError(
        "Expected exactly one historical reconstruct() method."
    )


loader_class_node, reconstruct_node = (
    reconstruct_candidates[
        0
    ]
)


HISTORICAL_LOADER_CLASS = (
    loader_class_node.name
)


if HISTORICAL_LOADER_CLASS != "Stage20CompactCorpus":

    raise RuntimeError(
        "Unexpected historical loader class."
    )


reconstruct_source = source_segment(
    loader_text,
    reconstruct_node,
)

reconstruct_source_sha = hashlib.sha256(
    reconstruct_source.encode(
        "utf-8"
    )
).hexdigest()


print(
    "Class:",
    HISTORICAL_LOADER_CLASS
)

print(
    "Method line:",
    reconstruct_node.lineno
)

print(
    "Method source SHA256:"
)

print(
    " ",
    reconstruct_source_sha
)


# -----------------------------------------------------------------------------
# Discover allocations inside reconstruct().
# -----------------------------------------------------------------------------

allocations = []


for node in ast.walk(
    reconstruct_node
):

    if not isinstance(
        node,
        (
            ast.Assign,
            ast.AnnAssign,
        ),
    ):

        continue


    value = getattr(
        node,
        "value",
        None,
    )


    if not isinstance(
        value,
        ast.Call,
    ):

        continue


    function_name = call_name(
        value.func
    )


    if function_name not in {
        "np.zeros",
        "numpy.zeros",
        "np.empty",
        "numpy.empty",
        "np.full",
        "numpy.full",
    }:

        continue


    if not value.args:

        continue


    target = None


    if isinstance(
        node,
        ast.Assign,
    ):

        if (
            len(
                node.targets
            )
            ==
            1
            and
            isinstance(
                node.targets[
                    0
                ],
                ast.Name,
            )
        ):

            target = node.targets[
                0
            ].id


    elif isinstance(
        node,
        ast.AnnAssign,
    ):

        if isinstance(
            node.target,
            ast.Name,
        ):

            target = node.target.id


    allocations.append(
        {
            "target":
                target,

            "function":
                function_name,

            "shape":
                shape_from_ast(
                    value.args[
                        0
                    ]
                ),

            "shape_source":
                source_segment(
                    loader_text,
                    value.args[
                        0
                    ],
                ),

            "dtype":
                dtype_from_call(
                    value
                ),

            "source":
                source_segment(
                    loader_text,
                    node,
                ),
        }
    )


print(
    "\nArray allocations inside reconstruct():"
)


for row in allocations:

    print(
        " ",
        row
    )


image_candidates = [
    row
    for row in allocations
    if (
        row[
            "shape"
        ]
        ==
        [
            ROWS,
            COLS,
        ]
        and
        row[
            "dtype"
        ]
        in {
            "np.uint8",
            "numpy.uint8",
            "uint8",
        }
    )
]


mask_candidates = [
    row
    for row in allocations
    if (
        row[
            "shape"
        ]
        ==
        [
            ROWS
        ]
        and
        row[
            "dtype"
        ]
        in {
            "bool",
            "np.bool_",
            "numpy.bool_",
            "np.bool",
            "numpy.bool",
        }
    )
]


if len(
    image_candidates
) != 1:

    raise RuntimeError(
        "Could not uniquely derive historical uint8 [64,256] image allocation."
    )


if len(
    mask_candidates
) != 1:

    raise RuntimeError(
        "Could not uniquely derive historical bool [64] packet-mask allocation."
    )


IMAGE_VARIABLE = (
    image_candidates[
        0
    ][
        "target"
    ]
)

MASK_VARIABLE = (
    mask_candidates[
        0
    ][
        "target"
    ]
)


# -----------------------------------------------------------------------------
# Verify both are returned by reconstruct().
# -----------------------------------------------------------------------------

return_nodes = [
    node
    for node in ast.walk(
        reconstruct_node
    )
    if isinstance(
        node,
        ast.Return,
    )
]


returned_names = set()


for node in return_nodes:

    value = node.value


    if isinstance(
        value,
        (
            ast.Tuple,
            ast.List,
        ),
    ):

        for element in value.elts:

            if isinstance(
                element,
                ast.Name,
            ):

                returned_names.add(
                    element.id
                )


    elif isinstance(
        value,
        ast.Name,
    ):

        returned_names.add(
            value.id
        )


print(
    "\nReturned local variables:",
    sorted(
        returned_names
    )
)


if IMAGE_VARIABLE not in returned_names:

    raise RuntimeError(
        "Derived image allocation is not returned by historical reconstruct()."
    )


if MASK_VARIABLE not in returned_names:

    raise RuntimeError(
        "Derived packet mask is not returned by historical reconstruct()."
    )


print(
    "\n[PASS] Historical representation contract derived:"
)

print(
    "  image:",
    IMAGE_VARIABLE,
    "uint8 [64,256]"
)

print(
    "  mask :",
    MASK_VARIABLE,
    "bool [64]"
)


# =============================================================================
# 8. CREATE LOCK DIRECTORY FIRST
# =============================================================================

LOCK_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# Preserve the transient recovery inventory byte-for-byte in Git.
shutil.copy2(
    INVENTORY_SOURCE,
    INVENTORY_COPY,
)


if sha256_file(
    INVENTORY_COPY
) != EXPECTED_INVENTORY_SHA256:

    raise RuntimeError(
        "Copied 4C0 inventory changed."
    )


# =============================================================================
# 9. REPRESENTATION BOUNDARY MAP
# =============================================================================

banner(
    "STAGE26-4C1-FREEZE-GIT :: FREEZE COMPONENT BOUNDARY"
)


boundary = {
    "schema":
        "stage26_representation_boundary_map_v2",

    "scientific_parent":
        EXPECTED_PARENT,

    "group":
        "GROUP_B_PACKET_IMAGE",

    "component_name":
        "STAGE20_COMPACT_CORPUS_TO_DENSE_UINT8_IMAGE_AND_PACKET_VALIDITY_MASK",

    "historical_contract": {
        "loader_class":
            HISTORICAL_LOADER_CLASS,

        "reconstruct_source_sha256":
            reconstruct_source_sha,

        "image_variable":
            IMAGE_VARIABLE,

        "image_shape":
            [
                ROWS,
                COLS,
            ],

        "image_dtype":
            "uint8",

        "packet_mask_variable":
            MASK_VARIABLE,

        "packet_mask_shape":
            [
                ROWS
            ],

        "packet_mask_dtype":
            "bool",
    },

    "authoritative_input": {
        "release_tag":
            RELEASE_TAG,

        "release_asset":
            RELEASE_ASSET,

        "release_tar_sha256":
            EXPECTED_TAR_SHA256,

        "flow_count":
            FLOW_COUNT,

        "encoded_byte_count":
            ENCODED_BYTE_COUNT,

        "files_used_by_component": [
            "encoded_bytes.bin",
            "flow_offsets.npy",
            "packet_lengths.npy",
        ],

        "labels_used":
            False,
    },

    "output": {
        "image_shape_per_flow":
            [
                ROWS,
                COLS,
            ],

        "image_dtype":
            "uint8",

        "packet_validity_mask_shape_per_flow":
            [
                ROWS
            ],

        "packet_validity_mask_dtype":
            "bool",

        "batch_image_shape":
            "[B,64,256]",

        "batch_packet_mask_shape":
            "[B,64]",
    },

    "included_operations": [
        "read packet_lengths row",
        "read flow byte offsets",
        "allocate zero-filled uint8 batch image array",
        "allocate bool packet-validity-mask batch array",
        "copy encoded retained bytes into packet-row prefixes",
        "mark rows containing retained packets as valid",
        "validate offset delta equals sum(packet_lengths)",
    ],

    "excluded_operations": [
        "GitHub Release download",
        "Release TAR unpacking/restoration",
        "raw PCAP reading",
        "flow reconstruction",
        "supervised matching/filtering",
        "packet selection",
        "header masking",
        "packet truncation and compact encoding",
        "labels.npy access",
        "float32 conversion",
        "division by 255",
        "torch tensor conversion",
        "CNN forward",
        "ViT forward",
        "prediction thresholding",
    ],

    "model_scaling_boundary": {
        "encoder_function":
            "scale_for_model",

        "conversion":
            "np.float32",

        "denominator":
            "np.float32(255.0)",

        "included_in_representation_component":
            False,
    },

    "claim_boundary": {
        "compact_corpus_dense_materialization_cost":
            True,

        "raw_flow_to_packet_image_complete_cost":
            False,

        "packet_masking_encoding_cost_included":
            False,

        "group_A_70_feature_extraction_profiled":
            False,

        "complete_end_to_end_claim_allowed":
            False,

        "results_sample_specific":
            True,
    },
}


atomic_json(
    BOUNDARY_PATH,
    boundary,
)


boundary_sha = sha256_file(
    BOUNDARY_PATH
)


print(
    "Boundary SHA256:"
)

print(
    " ",
    boundary_sha
)


# =============================================================================
# 10. REPRESENTATION-ONLY WORKER
# =============================================================================

banner(
    "STAGE26-4C1-FREEZE-GIT :: FREEZE WORKER"
)


WORKER_TEXT = r'''#!/usr/bin/env python3
from __future__ import annotations

import os
import sys
import json
import time
import hashlib
import importlib.util
from pathlib import Path

import numpy as np


ROWS = 64
COLS = 256


def atomic_json(path: Path, obj) -> None:

    path = Path(path)
    tmp = Path(str(path) + ".tmp")

    with tmp.open("w", encoding="utf-8") as fh:

        json.dump(
            obj,
            fh,
            indent=2,
            sort_keys=True,
        )

        fh.write("\n")
        fh.flush()
        os.fsync(fh.fileno())

    os.replace(tmp, path)


class RepresentationSource:

    def __init__(self, corpus_dir: Path):

        self.corpus_dir = Path(corpus_dir)

        self.encoded_path = (
            self.corpus_dir
            / "encoded_bytes.bin"
        )

        self.lengths_path = (
            self.corpus_dir
            / "packet_lengths.npy"
        )

        self.offsets_path = (
            self.corpus_dir
            / "flow_offsets.npy"
        )

        for path in (
            self.encoded_path,
            self.lengths_path,
            self.offsets_path,
        ):

            if not path.is_file():
                raise FileNotFoundError(path)

        self.packet_lengths = np.load(
            self.lengths_path,
            mmap_mode="r",
            allow_pickle=False,
        )

        self.flow_offsets = np.load(
            self.offsets_path,
            mmap_mode="r",
            allow_pickle=False,
        )

        self.encoded_bytes = np.memmap(
            self.encoded_path,
            dtype=np.uint8,
            mode="r",
        )

        if (
            self.packet_lengths.ndim != 2
            or
            self.packet_lengths.shape[1] != ROWS
        ):
            raise ValueError(
                "packet_lengths.npy must have shape [N,64]"
            )

        self.n = int(
            self.packet_lengths.shape[0]
        )

        if self.flow_offsets.shape != (
            self.n + 1,
        ):
            raise ValueError(
                "flow_offsets.npy must have shape [N+1]"
            )

        if self.packet_lengths.dtype != np.uint16:
            raise ValueError(
                "packet_lengths.npy must be uint16"
            )

        if self.flow_offsets.dtype != np.uint64:
            raise ValueError(
                "flow_offsets.npy must be uint64"
            )

        if int(self.flow_offsets[0]) != 0:
            raise ValueError(
                "first flow offset must be zero"
            )

        if (
            int(self.flow_offsets[-1])
            !=
            int(self.encoded_bytes.size)
        ):
            raise ValueError(
                "final flow offset does not equal encoded byte count"
            )


    def reconstruct(
        self,
        index: int,
    ) -> tuple[np.ndarray, np.ndarray]:

        index = int(index)

        if index < 0:
            index += self.n

        if index < 0 or index >= self.n:
            raise IndexError(index)

        lengths = np.asarray(
            self.packet_lengths[index],
            dtype=np.uint16,
        )

        if np.any(
            lengths > COLS
        ):
            raise ValueError(
                "packet length exceeds frozen 256-byte width"
            )

        # Frozen corpus geometry requires all retained packet rows
        # to precede zero-padding rows.
        positive = lengths > 0

        if np.any(
            positive[
                1:
            ]
            &
            ~np.maximum.accumulate(
                positive
            )[
                :-1
            ]
        ):
            raise ValueError(
                "non-contiguous retained packet rows"
            )

        start = int(
            self.flow_offsets[index]
        )

        end = int(
            self.flow_offsets[index + 1]
        )

        expected = int(
            lengths.astype(
                np.uint64
            ).sum()
        )

        if end - start != expected:
            raise ValueError(
                "offset delta does not equal sum(packet_lengths)"
            )

        image = np.zeros(
            (
                ROWS,
                COLS,
            ),
            dtype=np.uint8,
        )

        packet_mask = np.zeros(
            (
                ROWS,
            ),
            dtype=bool,
        )

        cursor = start

        for row_index, length in enumerate(
            lengths.tolist()
        ):

            length = int(length)

            if length == 0:
                continue

            next_cursor = (
                cursor
                +
                length
            )

            image[
                row_index,
                :length,
            ] = self.encoded_bytes[
                cursor:
                next_cursor
            ]

            packet_mask[
                row_index
            ] = True

            cursor = next_cursor

        if cursor != end:
            raise ValueError(
                "flow traversal ended at unexpected byte offset"
            )

        return (
            image,
            packet_mask,
        )


    def materialize_batch(
        self,
        indices: np.ndarray,
    ) -> tuple[np.ndarray, np.ndarray]:

        indices = np.asarray(
            indices,
            dtype=np.int64,
        ).reshape(-1)

        batch_size = int(
            indices.size
        )

        images = np.zeros(
            (
                batch_size,
                ROWS,
                COLS,
            ),
            dtype=np.uint8,
        )

        packet_masks = np.zeros(
            (
                batch_size,
                ROWS,
            ),
            dtype=bool,
        )

        for batch_index, flow_index in enumerate(
            indices.tolist()
        ):

            image, packet_mask = self.reconstruct(
                int(flow_index)
            )

            images[
                batch_index
            ] = image

            packet_masks[
                batch_index
            ] = packet_mask

        return (
            images,
            packet_masks,
        )


def load_module_from_path(
    name: str,
    path: Path,
):

    spec = importlib.util.spec_from_file_location(
        name,
        path,
    )

    if spec is None or spec.loader is None:
        raise RuntimeError(
            f"Could not import {path}"
        )

    module = importlib.util.module_from_spec(
        spec
    )

    spec.loader.exec_module(
        module
    )

    return module


def representation_fingerprint(
    image: np.ndarray,
    packet_mask: np.ndarray,
) -> str:

    h = hashlib.sha256()

    h.update(
        np.ascontiguousarray(
            image
        ).tobytes()
    )

    h.update(
        np.ascontiguousarray(
            packet_mask
        ).tobytes()
    )

    return h.hexdigest()


def run_equivalence(
    cfg: dict,
    source: RepresentationSource,
) -> dict:

    loader_module = load_module_from_path(
        "stage26_historical_compact_loader",
        Path(
            cfg[
                "historical_loader_path"
            ]
        ),
    )

    loader_class = getattr(
        loader_module,
        cfg[
            "historical_loader_class"
        ],
    )

    historical = loader_class(
        Path(
            cfg[
                "corpus_dir"
            ]
        )
    )

    start_index = int(
        cfg[
            "start_index"
        ]
    )

    flow_count = int(
        cfg[
            "flow_count"
        ]
    )

    aggregate = hashlib.sha256()

    for index in range(
        start_index,
        start_index + flow_count,
    ):

        new_image, new_mask = source.reconstruct(
            index
        )

        historical_result = historical.reconstruct(
            index
        )

        if not isinstance(
            historical_result,
            tuple,
        ):

            raise RuntimeError(
                "historical reconstruct() did not return tuple"
            )

        if len(
            historical_result
        ) < 2:

            raise RuntimeError(
                "historical reconstruct() returned fewer than two objects"
            )

        old_image = np.asarray(
            historical_result[
                0
            ]
        )

        old_mask = np.asarray(
            historical_result[
                1
            ]
        )

        if old_image.shape != (
            ROWS,
            COLS,
        ):
            raise RuntimeError(
                f"historical image shape mismatch at {index}"
            )

        if old_mask.shape != (
            ROWS,
        ):
            raise RuntimeError(
                f"historical packet mask shape mismatch at {index}"
            )

        if old_image.dtype != np.uint8:
            raise RuntimeError(
                f"historical image dtype mismatch at {index}"
            )

        if old_mask.dtype != np.bool_:
            raise RuntimeError(
                f"historical packet mask dtype mismatch at {index}"
            )

        if not np.array_equal(
            new_image,
            old_image,
        ):
            raise RuntimeError(
                f"image equivalence failure at flow {index}"
            )

        if not np.array_equal(
            new_mask,
            old_mask,
        ):
            raise RuntimeError(
                f"packet-mask equivalence failure at flow {index}"
            )

        aggregate.update(
            np.ascontiguousarray(
                new_image
            ).tobytes()
        )

        aggregate.update(
            np.ascontiguousarray(
                new_mask
            ).tobytes()
        )

    return {
        "status":
            "PASS",

        "mode":
            "VALIDATE_EQUIVALENCE",

        "start_index":
            start_index,

        "flow_count":
            flow_count,

        "end_index_exclusive":
            start_index + flow_count,

        "representation_fingerprint_sha256":
            aggregate.hexdigest(),

        "timing_performed":
            False,

        "labels_accessed":
            False,

        "model_loaded":
            False,

        "gpu_used":
            False,
    }


def run_benchmark(
    cfg: dict,
    source: RepresentationSource,
) -> dict:

    start_index = int(
        cfg[
            "start_index"
        ]
    )

    batch_size = int(
        cfg[
            "batch_size"
        ]
    )

    warmup_runs = int(
        cfg[
            "warmup_runs"
        ]
    )

    timed_runs = int(
        cfg[
            "timed_runs"
        ]
    )

    indices = np.arange(
        start_index,
        start_index + batch_size,
        dtype=np.int64,
    )

    warm_fingerprint = None

    for _ in range(
        warmup_runs
    ):

        images, packet_masks = source.materialize_batch(
            indices
        )

        warm_fingerprint = representation_fingerprint(
            images,
            packet_masks,
        )

    observations = []

    reference_fingerprint = None

    for iteration in range(
        1,
        timed_runs + 1,
    ):

        start_ns = time.perf_counter_ns()

        images, packet_masks = source.materialize_batch(
            indices
        )

        end_ns = time.perf_counter_ns()

        elapsed_ns = int(
            end_ns - start_ns
        )

        if elapsed_ns <= 0:
            raise RuntimeError(
                "non-positive elapsed time"
            )

        # Fingerprinting is intentionally outside the timer.
        fingerprint = representation_fingerprint(
            images,
            packet_masks,
        )

        if reference_fingerprint is None:

            reference_fingerprint = fingerprint

        elif fingerprint != reference_fingerprint:

            raise RuntimeError(
                "representation output changed across timed iterations"
            )

        elapsed_seconds = (
            elapsed_ns
            /
            1_000_000_000.0
        )

        observations.append(
            {
                "iteration_index":
                    iteration,

                "elapsed_ns":
                    elapsed_ns,

                "elapsed_seconds":
                    elapsed_seconds,

                "flows":
                    batch_size,

                "flows_per_second":
                    (
                        batch_size
                        /
                        elapsed_seconds
                    ),

                "image_output_bytes":
                    int(
                        images.nbytes
                    ),

                "packet_mask_output_bytes":
                    int(
                        packet_masks.nbytes
                    ),

                "total_output_bytes":
                    int(
                        images.nbytes
                        +
                        packet_masks.nbytes
                    ),
            }
        )

    return {
        "status":
            "PASS",

        "mode":
            "BENCHMARK_TIMED",

        "start_index":
            start_index,

        "batch_size":
            batch_size,

        "end_index_exclusive":
            start_index + batch_size,

        "warmup_runs":
            warmup_runs,

        "timed_runs":
            timed_runs,

        "representation_fingerprint_sha256":
            reference_fingerprint,

        "warm_representation_fingerprint_sha256":
            warm_fingerprint,

        "observations":
            observations,

        "timing_performed":
            True,

        "labels_accessed":
            False,

        "float32_div255_performed":
            False,

        "model_loaded":
            False,

        "gpu_used":
            False,
    }


def main():

    if len(
        sys.argv
    ) != 2:

        raise SystemExit(
            "usage: stage26_representation_worker_v1.py CONFIG.json"
        )

    config_path = Path(
        sys.argv[
            1
        ]
    )

    cfg = json.loads(
        config_path.read_text(
            encoding="utf-8"
        )
    )

    affinity = cfg.get(
        "affinity"
    )

    if affinity is not None:

        if not hasattr(
            os,
            "sched_setaffinity",
        ):

            raise RuntimeError(
                "sched_setaffinity unavailable"
            )

        os.sched_setaffinity(
            0,
            {
                int(cpu)
                for cpu in affinity
            },
        )

    source = RepresentationSource(
        Path(
            cfg[
                "corpus_dir"
            ]
        )
    )

    if source.n != int(
        cfg[
            "expected_flow_count"
        ]
    ):

        raise RuntimeError(
            "restored corpus flow count mismatch"
        )

    mode = cfg[
        "mode"
    ]

    if mode == "VALIDATE_EQUIVALENCE":

        result = run_equivalence(
            cfg,
            source,
        )

    elif mode == "BENCHMARK_TIMED":

        result = run_benchmark(
            cfg,
            source,
        )

    else:

        raise RuntimeError(
            f"unknown mode: {mode}"
        )

    result.update(
        {
            "schema":
                "stage26_representation_worker_result_v2",

            "worker_pid":
                os.getpid(),

            "population_flow_count":
                source.n,

            "image_dtype":
                "uint8",

            "image_shape_per_flow":
                [
                    ROWS,
                    COLS,
                ],

            "packet_mask_dtype":
                "bool",

            "packet_mask_shape_per_flow":
                [
                    ROWS
                ],

            "labels_file_opened":
                False,

            "corpus_created_or_modified":
                False,

            "pcap_accessed":
                False,
        }
    )

    atomic_json(
        Path(
            cfg[
                "result_path"
            ]
        ),
        result,
    )


if __name__ == "__main__":
    main()
'''


# In-memory syntax compile only.
# This deliberately avoids creating __pycache__ in the Git worktree.
compile(
    WORKER_TEXT,
    str(
        WORKER_PATH
    ),
    "exec",
)


atomic_text(
    WORKER_PATH,
    WORKER_TEXT,
)


worker_sha = sha256_file(
    WORKER_PATH
)


print(
    "Worker syntax: PASS"
)

print(
    "Worker SHA256:"
)

print(
    " ",
    worker_sha
)


# Static import audit.
worker_tree = ast.parse(
    WORKER_TEXT
)


import_roots = set()


for node in ast.walk(
    worker_tree
):

    if isinstance(
        node,
        ast.Import,
    ):

        for alias in node.names:

            import_roots.add(
                alias.name.split(
                    "."
                )[
                    0
                ]
            )


    elif isinstance(
        node,
        ast.ImportFrom,
    ):

        if node.module:

            import_roots.add(
                node.module.split(
                    "."
                )[
                    0
                ]
            )


for forbidden_module in {
    "scapy",
    "dpkt",
    "pyshark",
    "torch",
}:

    if forbidden_module in import_roots:

        raise RuntimeError(
            f"Forbidden worker import: {forbidden_module}"
        )


if (
    '"labels.npy"' in WORKER_TEXT
    or
    "'labels.npy'" in WORKER_TEXT
):

    raise RuntimeError(
        "Representation worker constructs or names labels.npy."
    )


print(
    "Worker forbidden-input audit: PASS"
)


# =============================================================================
# 11. FREEZE PROTOCOL
# =============================================================================

banner(
    "STAGE26-4C1-FREEZE-GIT :: FREEZE PROTOCOL"
)


protocol = {
    "schema":
        "stage26_representation_protocol_v2",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4C1",

    "status":
        "FROZEN_BEFORE_FIRST_REPRESENTATION_MEASUREMENT",

    "scientific_parent":
        EXPECTED_PARENT,

    "selection_independence": {
        "sample_selected_using_inference_results":
            False,

        "sample_selected_using_representation_timing":
            False,

        "sample_selected_using_labels":
            False,

        "seed":
            SEED,

        "selection_rule":
            (
                "Existing Stage26 seed 26042 modulo valid start positions "
                "for the largest frozen batch."
            ),
    },

    "authoritative_source": {
        "release_tag":
            RELEASE_TAG,

        "release_asset":
            RELEASE_ASSET,

        "release_tar_bytes":
            EXPECTED_TAR_BYTES,

        "release_tar_sha256":
            EXPECTED_TAR_SHA256,

        "population_flow_count":
            FLOW_COUNT,

        "member_identities":
            MEMBER_IDENTITIES,

        "restore_before_equivalence":
            True,

        "restore_operation_timed":
            False,

        "restore_rule":
            (
                "Extract exact already-published Release members byte-for-byte "
                "into isolated Stage26 runtime storage and verify each member "
                "against frozen SHA256. This is restoration, not regeneration."
            ),
    },

    "implementation": {
        "fresh_4c0_inventory_path":
            str(
                INVENTORY_COPY.relative_to(
                    REPO
                )
            ),

        "fresh_4c0_inventory_sha256":
            EXPECTED_INVENTORY_SHA256,

        "historical_encoder_path":
            str(
                ENCODER.relative_to(
                    REPO
                )
            ),

        "historical_encoder_sha256":
            EXPECTED_ENCODER_SHA256,

        "historical_loader_path":
            str(
                LOADER.relative_to(
                    REPO
                )
            ),

        "historical_loader_sha256":
            EXPECTED_LOADER_SHA256,

        "historical_loader_class":
            HISTORICAL_LOADER_CLASS,

        "historical_reconstruct_source_sha256":
            reconstruct_source_sha,

        "worker_path":
            str(
                WORKER_PATH.relative_to(
                    REPO
                )
            ),

        "worker_sha256":
            worker_sha,

        "boundary_map_path":
            str(
                BOUNDARY_PATH.relative_to(
                    REPO
                )
            ),

        "boundary_map_sha256":
            boundary_sha,
    },

    "historical_output_contract": {
        "image_shape":
            [
                ROWS,
                COLS,
            ],

        "image_dtype":
            "uint8",

        "packet_mask_shape":
            [
                ROWS
            ],

        "packet_mask_dtype":
            "bool",
    },

    "mandatory_pre_timing_equivalence_gate": {
        "required":
            True,

        "start_index_zero_based":
            EQUIVALENCE_START,

        "flow_count":
            EQUIVALENCE_FLOW_COUNT,

        "comparison":
            (
                "For every gated flow, worker image and packet mask must be "
                "exactly np.array_equal to frozen "
                "Stage20CompactCorpus.reconstruct() outputs."
            ),

        "timing_during_equivalence":
            False,

        "timing_allowed_before_gate_pass":
            False,

        "gate_result_must_be_git_anchored_before_timing":
            True,
    },

    "measurement_boundary": {
        "input":
            "RESTORED_EXACT_STAGE20_COMPACT_RELEASE_FILES",

        "output":
            "UINT8_IMAGE_BATCH_AND_BOOL_PACKET_MASK_BATCH",

        "batch_image_shape":
            "[B,64,256]",

        "batch_packet_mask_shape":
            "[B,64]",

        "batch_output_allocation_included":
            True,

        "memmap_reads_included":
            True,

        "release_download_included":
            False,

        "release_tar_unpacking_included":
            False,

        "loader_initialization_included":
            False,

        "Python_process_startup_included":
            False,

        "JSON_serialization_included":
            False,

        "fingerprinting_included":
            False,

        "labels_accessed":
            False,

        "float32_conversion_included":
            False,

        "division_by_255_included":
            False,

        "model_forward_included":
            False,

        "page_cache_manipulated":
            False,

        "warm_exact_batch_before_timing":
            True,

        "cold_disk_IO_claim_allowed":
            False,

        "interpretation":
            (
                "Warm compact-corpus dense-materialization cost under existing "
                "OS page-cache state; not cold-storage performance."
            ),
    },

    "sample": {
        "population_flow_count":
            FLOW_COUNT,

        "sampling_rule":
            "DETERMINISTIC_NESTED_CONTIGUOUS_SEGMENT",

        "start_index_zero_based":
            SAMPLE_START,

        "largest_end_index_zero_based_exclusive":
            SAMPLE_END_EXCLUSIVE,

        "largest_batch_flow_count":
            MAX_BATCH_SIZE,

        "largest_batch_population_fraction":
            (
                MAX_BATCH_SIZE
                /
                FLOW_COUNT
            ),

        "batch_sizes":
            BATCH_SIZES,

        "nested_rule":
            (
                "For batch B use flows [start,start+B); every smaller "
                "condition is a prefix of the largest frozen sample."
            ),

        "sample_specific_claim_required":
            True,
    },

    "hardware": {
        "mode":
            "CPU_1_PHYSICAL_CORE",

        "affinity":
            CPU_AFFINITY,

        "thread_count":
            THREAD_COUNT,

        "gpu":
            False,
    },

    "iteration_policy": {
        str(
            batch
        ):
            ITERATION_POLICY[
                batch
            ]
        for batch in BATCH_SIZES
    },

    "condition_count":
        len(
            BATCH_SIZES
        ),

    "fresh_process_per_batch_condition":
        True,

    "condition_timeout_seconds":
        CONDITION_TIMEOUT_SECONDS,

    "environment_gate":
        ENVIRONMENT_GATE,

    "failure_policy": {
        "INVALID_ENVIRONMENT":
            "Do not benchmark condition.",

        "TIMEOUT_RESOURCE_LIMIT":
            "Record timeout; do not adapt protocol.",

        "RESOURCE_LIMIT_OOM":
            "Record resource limit; do not reduce batch.",

        "IMPLEMENTATION_FAILURE":
            (
                "Stop for narrow source-faithful diagnosis; "
                "do not alter sample or timing policy."
            ),

        "no_post_result_adaptation":
            True,
    },

    "required_raw_metrics": [
        "batch_size",
        "iteration_index",
        "elapsed_ns",
        "elapsed_seconds",
        "flows_per_second",
        "image_output_bytes",
        "packet_mask_output_bytes",
        "total_output_bytes",
    ],

    "summary_metrics": [
        "p50_batch_latency",
        "p95_batch_latency",
        "p99_batch_latency_when_n_gte_100",
        "median_flows_per_second",
        "minimum_flows_per_second",
        "maximum_flows_per_second",
    ],

    "claim_boundary": {
        "component_name":
            (
                "Stage20 compact Release corpus -> dense uint8 image "
                "+ bool packet-validity mask"
            ),

        "applies_to":
            "GROUP_B_PACKET_IMAGE",

        "applies_to_group_A_70_feature_models":
            False,

        "raw_flow_to_packet_image_complete":
            False,

        "packet_masking_encoding_cost_included":
            False,

        "complete_pipeline_additivity_claim_allowed":
            False,

        "representation_results_sample_specific":
            True,
    },

    "scientific_rules": {
        "release_corpus_authoritative":
            True,

        "release_corpus_regeneration_forbidden":
            True,

        "pcap_may_be_opened":
            False,

        "labels_may_be_opened":
            False,

        "models_may_be_loaded":
            False,

        "Thursday_may_be_accessed":
            False,

        "Friday_may_be_accessed":
            False,

        "GPU_used":
            False,

        "first_representation_timing_allowed_only_after":
            (
                "this protocol is remotely Git-anchored, exact Release "
                "members are restored and verified, the frozen 128-flow "
                "untimed equivalence gate passes, and that equivalence "
                "result is itself remotely Git-anchored."
            ),
    },
}


atomic_json(
    PROTOCOL_PATH,
    protocol,
)


protocol_sha = sha256_file(
    PROTOCOL_PATH
)


print(
    "Protocol SHA256:"
)

print(
    " ",
    protocol_sha
)


print(
    "\nFrozen sample:"
)

print(
    "  start       :",
    SAMPLE_START
)

print(
    "  max end     :",
    SAMPLE_END_EXCLUSIVE
)

print(
    "  max batch   :",
    MAX_BATCH_SIZE
)

print(
    "  population  :",
    FLOW_COUNT
)


# =============================================================================
# 12. UPSTREAM PROVENANCE
# =============================================================================

provenance = {
    "schema":
        "stage26_4c_upstream_provenance_v2",

    "scientific_parent":
        EXPECTED_PARENT,

    "fresh_recovery_inventory": {
        "repo_relative_path":
            str(
                INVENTORY_COPY.relative_to(
                    REPO
                )
            ),

        "sha256":
            EXPECTED_INVENTORY_SHA256,
    },

    "stage26_raw_extraction": {
        "manifest_repo_relative_path":
            str(
                RAW_TIMING_MANIFEST.relative_to(
                    REPO
                )
            ),

        "manifest_sha256":
            raw_manifest_sha,

        "closed":
            True,
    },

    "stage20": {
        "encoder": {
            "repo_relative_path":
                str(
                    ENCODER.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_ENCODER_SHA256,
        },

        "compact_loader": {
            "repo_relative_path":
                str(
                    LOADER.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_LOADER_SHA256,

            "class":
                HISTORICAL_LOADER_CLASS,

            "reconstruct_source_sha256":
                reconstruct_source_sha,

            "derived_output_contract": {
                "image_shape":
                    [
                        ROWS,
                        COLS,
                    ],

                "image_dtype":
                    "uint8",

                "packet_mask_shape":
                    [
                        ROWS
                    ],

                "packet_mask_dtype":
                    "bool",
            },
        },

        "architecture_lock": {
            "repo_relative_path":
                str(
                    ARCH_LOCK.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_ARCH_LOCK_SHA256,
        },
    },

    "stage26_measurement_protocol": {
        "repo_relative_path":
            str(
                MEASUREMENT_PROTOCOL.relative_to(
                    REPO
                )
            ),

        "sha256":
            EXPECTED_MEASUREMENT_PROTOCOL_SHA256,
    },

    "release": {
        "tag":
            RELEASE_TAG,

        "asset":
            RELEASE_ASSET,

        "tar_bytes":
            EXPECTED_TAR_BYTES,

        "tar_sha256":
            EXPECTED_TAR_SHA256,

        "members":
            MEMBER_IDENTITIES,
    },

    "geometry": {
        "flow_count":
            FLOW_COUNT,

        "rows":
            ROWS,

        "cols":
            COLS,

        "encoded_byte_count":
            ENCODED_BYTE_COUNT,
    },
}


atomic_json(
    PROVENANCE_PATH,
    provenance,
)


provenance_sha = sha256_file(
    PROVENANCE_PATH
)


# =============================================================================
# 13. FREEZE RECEIPT
# =============================================================================

freeze_receipt = {
    "schema":
        "stage26_4c1_representation_protocol_freeze_receipt_v2",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4C1",

    "status":
        "REPRESENTATION_PROTOCOL_FROZEN_NOT_YET_EXECUTED",

    "scientific_parent":
        EXPECTED_PARENT,

    "fresh_4c0_inventory_sha256":
        EXPECTED_INVENTORY_SHA256,

    "worker_sha256":
        worker_sha,

    "protocol_sha256":
        protocol_sha,

    "boundary_map_sha256":
        boundary_sha,

    "upstream_provenance_sha256":
        provenance_sha,

    "historical_contract": {
        "loader_class":
            HISTORICAL_LOADER_CLASS,

        "reconstruct_source_sha256":
            reconstruct_source_sha,

        "image_shape":
            [
                ROWS,
                COLS,
            ],

        "image_dtype":
            "uint8",

        "packet_mask_shape":
            [
                ROWS
            ],

        "packet_mask_dtype":
            "bool",
    },

    "frozen_condition": {
        "CPU":
            "CPU_1_PHYSICAL_CORE",

        "affinity":
            CPU_AFFINITY,

        "thread_count":
            THREAD_COUNT,

        "batch_sizes":
            BATCH_SIZES,

        "sample_start_index_zero_based":
            SAMPLE_START,

        "largest_sample_end_index_exclusive":
            SAMPLE_END_EXCLUSIVE,

        "seed":
            SEED,
    },

    "mandatory_pre_timing_gate": {
        "equivalence_required":
            True,

        "flow_count":
            EQUIVALENCE_FLOW_COUNT,

        "start_index_zero_based":
            EQUIVALENCE_START,

        "must_be_git_anchored_before_timing":
            True,
    },

    "scientific_state": {
        "release_tar_extracted":
            False,

        "dense_representation_materialized":
            False,

        "equivalence_executed":
            False,

        "representation_timing_performed":
            False,

        "pcap_accessed":
            False,

        "release_corpus_regenerated":
            False,

        "labels_accessed":
            False,

        "models_loaded":
            False,

        "inference_performed":
            False,

        "Thursday_accessed":
            False,

        "Friday_accessed":
            False,

        "gpu_used":
            False,
    },
}


atomic_json(
    FREEZE_RECEIPT_PATH,
    freeze_receipt,
)


freeze_receipt_sha = sha256_file(
    FREEZE_RECEIPT_PATH
)


# =============================================================================
# 14. LOCK MANIFEST
# =============================================================================

banner(
    "STAGE26-4C1-FREEZE-GIT :: BUILD LOCK MANIFEST"
)


package_files = [
    INVENTORY_COPY,
    WORKER_PATH,
    PROTOCOL_PATH,
    BOUNDARY_PATH,
    PROVENANCE_PATH,
    FREEZE_RECEIPT_PATH,
]


manifest_rows = []


for path in package_files:

    manifest_rows.append(
        {
            "repo_relative_path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest = {
    "schema":
        "stage26_4c_representation_lock_manifest_v2",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        "READY_FOR_GIT_ANCHOR",

    "parent_commit":
        EXPECTED_PARENT,

    "file_count_excluding_manifest":
        len(
            manifest_rows
        ),

    "fresh_4c0_inventory_sha256":
        EXPECTED_INVENTORY_SHA256,

    "worker_sha256":
        worker_sha,

    "protocol_sha256":
        protocol_sha,

    "boundary_sha256":
        boundary_sha,

    "provenance_sha256":
        provenance_sha,

    "freeze_receipt_sha256":
        freeze_receipt_sha,

    "files":
        manifest_rows,
}


atomic_json(
    MANIFEST_PATH,
    manifest,
)


manifest_sha = sha256_file(
    MANIFEST_PATH
)


print(
    "Inventory  :",
    EXPECTED_INVENTORY_SHA256
)

print(
    "Worker     :",
    worker_sha
)

print(
    "Protocol   :",
    protocol_sha
)

print(
    "Boundary   :",
    boundary_sha
)

print(
    "Provenance :",
    provenance_sha
)

print(
    "Receipt    :",
    freeze_receipt_sha
)

print(
    "Manifest   :",
    manifest_sha
)


# =============================================================================
# 15. LOCAL PACKAGE SELF-AUDIT
# =============================================================================

banner(
    "STAGE26-4C1-FREEZE-GIT :: LOCAL PACKAGE AUDIT"
)


for row in manifest_rows:

    path = (
        REPO
        / row[
            "repo_relative_path"
        ]
    )

    size = int(
        path.stat().st_size
    )

    digest = sha256_file(
        path
    )

    passed = (
        size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        digest
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{size:10,d} B "
        f"{digest} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Local package audit failed."
        )


# =============================================================================
# 16. GIT CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-4C1-FREEZE-GIT :: GIT CHANGE AUDIT"
)


repo_status = git(
    "status",
    "--porcelain",
)


print(
    repo_status
)


if not repo_status:

    raise RuntimeError(
        "Expected uncommitted 4C1 lock package."
    )


unexpected = []


for line in repo_status.splitlines():

    rel = line[
        3:
    ]


    if not rel.startswith(
        str(
            LOCK_REL
        )
        +
        "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository changes:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 17. RESTORE REPO-LOCAL GIT IDENTITY
# =============================================================================

author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_PARENT,
)

author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_PARENT,
)


if (
    not author_name.strip()
    or
    "@"
    not in
    author_email
):

    raise RuntimeError(
        "Could not recover valid Git author identity."
    )


git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


print(
    "\nGit author:"
)

print(
    " ",
    author_name,
    "<" + author_email + ">"
)


# =============================================================================
# 18. COMMIT
# =============================================================================

banner(
    "STAGE26-4C1-FREEZE-GIT :: COMMIT"
)


git(
    "add",
    str(
        LOCK_REL
    ),
)


staged = git(
    "diff",
    "--cached",
    "--name-status",
)


print(
    staged
)


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "4C1 commit parent mismatch."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "4C1 commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository not clean after commit."
    )


# =============================================================================
# 19. PUSH
# =============================================================================

banner(
    "STAGE26-4C1-FREEZE-GIT :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    push = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
        check=True,
        text=True,
    )


    print(
        push.stdout.strip()
    )


github_token = None


# =============================================================================
# 20. REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-4C1-FREEZE-GIT :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "origin/main does not equal new 4C1 commit."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote subject mismatch."
    )


# =============================================================================
# 21. REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-4C1-FREEZE-GIT :: REMOTE BYTE VERIFICATION"
)


manifest_rel = str(
    MANIFEST_PATH.relative_to(
        REPO
    )
)


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    manifest_rel,
)

remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "Remote manifest SHA256:"
)

print(
    " ",
    remote_manifest_sha
)


if remote_manifest_sha != manifest_sha:

    raise RuntimeError(
        "Remote manifest mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


if remote_manifest[
    "file_count_excluding_manifest"
] != 6:

    raise RuntimeError(
        "Unexpected remote lock file count."
    )


for row in remote_manifest[
    "files"
]:

    data = git_blob_bytes(
        "origin/main",
        row[
            "repo_relative_path"
        ],
    )

    actual_size = len(
        data
    )

    actual_sha = sha256_bytes(
        data
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Remote byte verification failed."
        )


# =============================================================================
# 22. REMOTE SCIENTIFIC CONTENT AUDIT
# =============================================================================

banner(
    "STAGE26-4C1-FREEZE-GIT :: REMOTE SCIENTIFIC AUDIT"
)


remote_protocol = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            PROTOCOL_PATH.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_receipt = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            FREEZE_RECEIPT_PATH.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)


remote_checks = {
    "protocol_frozen":
        (
            remote_protocol[
                "status"
            ]
            ==
            "FROZEN_BEFORE_FIRST_REPRESENTATION_MEASUREMENT"
        ),

    "fresh_inventory":
        (
            remote_protocol[
                "implementation"
            ][
                "fresh_4c0_inventory_sha256"
            ]
            ==
            EXPECTED_INVENTORY_SHA256
        ),

    "image_64x256":
        (
            remote_protocol[
                "historical_output_contract"
            ][
                "image_shape"
            ]
            ==
            [
                64,
                256,
            ]
        ),

    "mask_64":
        (
            remote_protocol[
                "historical_output_contract"
            ][
                "packet_mask_shape"
            ]
            ==
            [
                64
            ]
        ),

    "equivalence_required":
        (
            remote_protocol[
                "mandatory_pre_timing_equivalence_gate"
            ][
                "required"
            ]
            is True
        ),

    "equivalence_not_timed":
        (
            remote_protocol[
                "mandatory_pre_timing_equivalence_gate"
            ][
                "timing_during_equivalence"
            ]
            is False
        ),

    "equivalence_anchor_required":
        (
            remote_protocol[
                "mandatory_pre_timing_equivalence_gate"
            ][
                "gate_result_must_be_git_anchored_before_timing"
            ]
            is True
        ),

    "div255_excluded":
        (
            remote_protocol[
                "measurement_boundary"
            ][
                "division_by_255_included"
            ]
            is False
        ),

    "labels_excluded":
        (
            remote_protocol[
                "measurement_boundary"
            ][
                "labels_accessed"
            ]
            is False
        ),

    "complete_E2E_blocked":
        (
            remote_protocol[
                "claim_boundary"
            ][
                "complete_pipeline_additivity_claim_allowed"
            ]
            is False
        ),

    "receipt_not_executed":
        (
            remote_receipt[
                "status"
            ]
            ==
            "REPRESENTATION_PROTOCOL_FROZEN_NOT_YET_EXECUTED"
        ),

    "timing_false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "representation_timing_performed"
            ]
            is False
        ),

    "equivalence_false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "equivalence_executed"
            ]
            is False
        ),

    "corpus_not_regenerated":
        (
            remote_receipt[
                "scientific_state"
            ][
                "release_corpus_regenerated"
            ]
            is False
        ),

    "GPU_false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "gpu_used"
            ]
            is False
        ),
}


for name, passed in remote_checks.items():

    print(
        f"{name:42s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    remote_checks.values()
):

    raise RuntimeError(
        "Remote 4C1 scientific audit failed."
    )


# =============================================================================
# 23. FINAL AUDIT
# =============================================================================

banner(
    "STAGE26-4C1-FREEZE-GIT :: FINAL AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != final_remote:

    raise RuntimeError(
        "Local/remote divergence."
    )


if final_status:

    raise RuntimeError(
        "Repository not clean after 4C1 anchor."
    )


# =============================================================================
# 24. CLOSURE
# =============================================================================

banner(
    "STAGE26-4C1 FREEZE + GIT ANCHOR COMPLETE"
)


print(
    "COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nPARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nFRESH 4C0 INVENTORY:"
)

print(
    " ",
    EXPECTED_INVENTORY_SHA256
)


print(
    "\nHISTORICAL OUTPUT CONTRACT:"
)

print(
    "  image        : uint8 [64,256]"
)

print(
    "  packet mask  : bool [64]"
)

print(
    "  loader class :",
    HISTORICAL_LOADER_CLASS
)

print(
    "  reconstruct source SHA256:"
)

print(
    "   ",
    reconstruct_source_sha
)


print(
    "\nFROZEN SAMPLE:"
)

print(
    "  seed        :",
    SEED
)

print(
    "  start       :",
    SAMPLE_START
)

print(
    "  largest end :",
    SAMPLE_END_EXCLUSIVE
)

print(
    "  batches     :",
    BATCH_SIZES
)


print(
    "\nLOCK HASHES:"
)

print(
    "  inventory  :",
    EXPECTED_INVENTORY_SHA256
)

print(
    "  worker     :",
    worker_sha
)

print(
    "  protocol   :",
    protocol_sha
)

print(
    "  boundary   :",
    boundary_sha
)

print(
    "  provenance :",
    provenance_sha
)

print(
    "  receipt    :",
    freeze_receipt_sha
)

print(
    "  manifest   :",
    manifest_sha
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  local == origin/main   : PASS"
)

print(
    "  parent                 : PASS"
)

print(
    "  six manifest files     : PASS"
)

print(
    "  manifest               : PASS"
)

print(
    "  scientific content     : PASS"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  REPRESENTATION PROTOCOL REMOTELY LOCKED : YES"
)

print(
    "  RELEASE TAR EXTRACTED                  : NO"
)

print(
    "  DENSE REPRESENTATION MATERIALIZED      : NO"
)

print(
    "  EQUIVALENCE EXECUTED                   : NO"
)

print(
    "  REPRESENTATION TIMING                  : NO"
)

print(
    "  PCAP ACCESSED                          : NO"
)

print(
    "  CORPUS REGENERATED                     : NO"
)

print(
    "  LABELS ACCESSED                        : NO"
)

print(
    "  MODELS LOADED                          : NO"
)

print(
    "  COMPLETE E2E CLAIM                     : NO"
)

print(
    "  GPU                                    : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Restore the three required Release members into an isolated"
)

print(
    "  runtime directory and execute the frozen 128-flow UNTIMED"
)

print(
    "  equivalence gate against Stage20CompactCorpus.reconstruct()."
)

print(
    "  Representation timing remains forbidden until that gate is"
)

print(
    "  separately committed and pushed."
)


STAGE26-4C1-FREEZE-GIT :: SCIENTIFIC PARENT
Expected parent: 55aba2e1cc08385659479c6275234dc23b11d231
Local HEAD     : 55aba2e1cc08385659479c6275234dc23b11d231
origin/main    : 55aba2e1cc08385659479c6275234dc23b11d231
Repo clean     : True

STAGE26-4C1-FREEZE-GIT :: FRESH 4C0 INVENTORY
Expected: b32a2a21407261d0c3628e80d1c39aab1b6921cf144b9ce5daea28392ec989a5
Actual  : b32a2a21407261d0c3628e80d1c39aab1b6921cf144b9ce5daea28392ec989a5
status_PASS                             : PASS
scientific_parent                       : PASS
release_restored                        : PASS
corpus_not_regenerated                  : PASS
representation_not_materialized         : PASS
equivalence_not_run                     : PASS
timing_not_run                          : PASS
pcap_not_accessed                       : PASS
gpu_not_used                            : PASS
release_sha                             : PASS

STAGE26-4C1-FREEZE-GIT :: UPSTREAM PROVENANCE
Stage20 encoder                  PASS 9883fe2

RuntimeError: Could not uniquely derive historical bool [64] packet-mask allocation.

In [6]:
# =============================================================================
# STAGE26-4C1-RECOVERY-GIT
# CORRECT HISTORICAL BYTE-LEVEL PADDING MASK CONTRACT
# FREEZE + COMMIT + PUSH IN ONE CELL
#
# LAST FAILED CELL ESTABLISHED FROM EXACT HASHED SOURCE:
#
#   Stage20CompactCorpus.reconstruct()
#
#       image
#           np.zeros((ROWS, COLS), dtype=np.uint8)
#           -> uint8 [64,256]
#
#       padding_mask
#           np.zeros((ROWS, COLS), dtype=np.bool_)
#           -> bool [64,256]
#
# PREVIOUS CELL FAILED ONLY BECAUSE IT EXPECTED bool [64].
#
# IMPORTANT:
#   Failure occurred BEFORE LOCK_DIR creation.
#
# THIS RECOVERY:
#   - re-verifies clean durable parent;
#   - re-derives historical contract from exact loader AST;
#   - freezes byte-level mask semantics [64,256];
#   - freezes corrected representation worker;
#   - creates the complete Stage26-4C1 lock package;
#   - commits + pushes immediately;
#   - remotely byte-verifies every artifact.
#
# NO:
#   - Release TAR extraction
#   - representation materialization
#   - equivalence execution
#   - representation timing
#   - PCAP
#   - corpus regeneration
#   - labels
#   - models
#   - inference
#   - GPU
# =============================================================================

from __future__ import annotations

import ast
import os
import json
import stat
import shutil
import hashlib
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. FROZEN IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

REP_RUNTIME_ROOT = (
    STAGE26_ROOT
    / "representation"
)

EXPECTED_PARENT = (
    "55aba2e1cc08385659479c6275234dc23b11d231"
)

COMMIT_SUBJECT = (
    "stage26: freeze representation profiling protocol"
)


# -----------------------------------------------------------------------------
# Fresh recovered 4C0 inventory
# -----------------------------------------------------------------------------

INVENTORY_SOURCE = (
    REP_RUNTIME_ROOT
    / "stage26_4c0_representation_implementation_inventory.json"
)

EXPECTED_INVENTORY_SHA256 = (
    "b32a2a21407261d0c3628e80d1c39aab1b6921cf144b9ce5daea28392ec989a5"
)


# -----------------------------------------------------------------------------
# Frozen Stage20 implementation
# -----------------------------------------------------------------------------

ENCODER = (
    REPO
    / "scripts"
    / "stage20_packet_image_encoder.py"
)

EXPECTED_ENCODER_SHA256 = (
    "9883fe2b27020aaff707a753123b35eb3223d21abf295d056ec233e532f94222"
)

LOADER = (
    REPO
    / "scripts"
    / "stage20_compact_corpus.py"
)

EXPECTED_LOADER_SHA256 = (
    "a1ba15881afeb1cf4de9225a06df9ae676b95f596c8ddced7734a445ba7624d0"
)

ARCH_LOCK = (
    REPO
    / "results"
    / "stage20_1e_training"
    / "stage20_1e0_architecture_training_protocol_lock.json"
)

EXPECTED_ARCH_LOCK_SHA256 = (
    "d3bba4d9d9df4432383a7f4239a8562f484cb5805b66d52794873bfaca837b3b"
)

MEASUREMENT_PROTOCOL = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXPECTED_MEASUREMENT_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)


# -----------------------------------------------------------------------------
# Closed Stage26-4B3 raw extraction
# -----------------------------------------------------------------------------

RAW_TIMING_MANIFEST = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4b3_cpu1_extraction_timing"
    / "stage26_4b3_cpu1_extraction_manifest.json"
)

EXPECTED_RAW_TIMING_MANIFEST_SHA256 = (
    "39900c940ca25f3a541efe0dcb5f08d51408581f1f6f4b93980b093327276bc1"
)


# -----------------------------------------------------------------------------
# Authoritative existing Monday Release corpus
# -----------------------------------------------------------------------------

MONDAY_TAR = (
    STAGE26_ROOT
    / "release_corpora"
    / "Monday"
    / "stage20-Monday-compact-corpus-v1.tar"
)

EXPECTED_TAR_BYTES = (
    595_261_440
)

EXPECTED_TAR_SHA256 = (
    "4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20"
)

RELEASE_TAG = (
    "stage20-compact-corpora-v1"
)

RELEASE_ASSET = (
    "stage20-Monday-compact-corpus-v1.tar"
)


MEMBER_IDENTITIES = {
    "encoded_bytes.bin": {
        "tar_member":
            "Monday/encoded_bytes.bin",

        "size_bytes":
            522_845_159,

        "sha256":
            "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",
    },

    "flow_offsets.npy": {
        "tar_member":
            "Monday/flow_offsets.npy",

        "size_bytes":
            4_228_208,

        "sha256":
            "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",
    },

    "labels.npy": {
        "tar_member":
            "Monday/labels.npy",

        "size_bytes":
            528_637,

        "sha256":
            "48792b8d6a127b35342cb0789baa6c54396f1100a60ce7225daf08d1c3530424",
    },

    "packet_lengths.npy": {
        "tar_member":
            "Monday/packet_lengths.npy",

        "size_bytes":
            67_649_280,

        "sha256":
            "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
    },
}


FLOW_COUNT = 528_509
ROWS = 64
COLS = 256
ENCODED_BYTE_COUNT = 522_845_159


# =============================================================================
# 1. FROZEN 4C MEASUREMENT POLICY
# =============================================================================

SEED = 26_042

BATCH_SIZES = [
    1,
    64,
    256,
    1024,
    8192,
]

MAX_BATCH_SIZE = max(
    BATCH_SIZES
)

SAMPLE_START = (
    SEED
    %
    (
        FLOW_COUNT
        -
        MAX_BATCH_SIZE
        +
        1
    )
)

SAMPLE_END_EXCLUSIVE = (
    SAMPLE_START
    +
    MAX_BATCH_SIZE
)


if SAMPLE_START != 26_042:

    raise RuntimeError(
        "Unexpected deterministic sample start."
    )


ITERATION_POLICY = {
    1: {
        "warmup_runs": 50,
        "timed_runs": 200,
        "role": "LATENCY_PRIMARY",
    },

    64: {
        "warmup_runs": 30,
        "timed_runs": 150,
        "role": "MICROBATCH",
    },

    256: {
        "warmup_runs": 20,
        "timed_runs": 100,
        "role": "STANDARD_BATCH",
    },

    1024: {
        "warmup_runs": 10,
        "timed_runs": 50,
        "role": "LARGE_BATCH",
    },

    8192: {
        "warmup_runs": 5,
        "timed_runs": 20,
        "role": "CAPACITY_BATCH",
    },
}


CPU_AFFINITY = [0]
THREAD_COUNT = 1
CONDITION_TIMEOUT_SECONDS = 600

EQUIVALENCE_START = SAMPLE_START
EQUIVALENCE_FLOW_COUNT = 128


ENVIRONMENT_GATE = {
    "cpu_utilization_percent_max":
        20.0,

    "available_ram_gib_min":
        8.0,

    "cpu_utilization_sampling_seconds":
        1.0,

    "retry_count":
        3,

    "retry_cooldown_seconds":
        5,

    "if_gate_never_passes":
        "INVALID_ENVIRONMENT",
}


# =============================================================================
# 2. LOCK PACKAGE
# =============================================================================

LOCK_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_4c_representation_protocol_lock"
)

LOCK_DIR = (
    REPO
    / LOCK_REL
)

INVENTORY_COPY = (
    LOCK_DIR
    / "stage26_4c0_representation_implementation_inventory.json"
)

WORKER_PATH = (
    LOCK_DIR
    / "stage26_representation_worker_v1.py"
)

PROTOCOL_PATH = (
    LOCK_DIR
    / "stage26_representation_protocol.json"
)

BOUNDARY_PATH = (
    LOCK_DIR
    / "stage26_representation_boundary_map.json"
)

PROVENANCE_PATH = (
    LOCK_DIR
    / "stage26_4c_upstream_provenance.json"
)

FREEZE_RECEIPT_PATH = (
    LOCK_DIR
    / "stage26_4c1_representation_protocol_freeze_receipt.json"
)

MANIFEST_PATH = (
    LOCK_DIR
    / "stage26_4c_representation_lock_manifest.json"
)


# =============================================================================
# 3. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 122
    )

    print(text)

    print(
        "=" * 122
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        check=True,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_text(
    path,
    text,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        f.write(
            text
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def atomic_json(
    path,
    obj,
):

    atomic_text(
        path,
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
        )
        +
        "\n",
    )


def source_segment(
    source,
    node,
):

    value = ast.get_source_segment(
        source,
        node,
    )

    return (
        value
        if value is not None
        else "<source unavailable>"
    )


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    result = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        check=True,
        text=False,
    )

    return bytes(
        result.stdout
    )


def call_name(node):

    if isinstance(
        node,
        ast.Name,
    ):

        return node.id


    if isinstance(
        node,
        ast.Attribute,
    ):

        left = call_name(
            node.value
        )

        if left:

            return (
                left
                +
                "."
                +
                node.attr
            )

        return node.attr


    return None


def resolve_shape(node):

    if isinstance(
        node,
        ast.Name,
    ):

        if node.id == "ROWS":
            return [64]

        if node.id == "COLS":
            return [256]


    if isinstance(
        node,
        ast.Constant,
    ) and isinstance(
        node.value,
        int,
    ):

        return [
            int(
                node.value
            )
        ]


    if isinstance(
        node,
        (
            ast.Tuple,
            ast.List,
        ),
    ):

        result = []


        for element in node.elts:

            if isinstance(
                element,
                ast.Name,
            ):

                if element.id == "ROWS":

                    result.append(
                        64
                    )

                elif element.id == "COLS":

                    result.append(
                        256
                    )

                else:

                    return None


            elif isinstance(
                element,
                ast.Constant,
            ) and isinstance(
                element.value,
                int,
            ):

                result.append(
                    int(
                        element.value
                    )
                )


            else:

                return None


        return result


    return None


def resolve_dtype(call):

    if not isinstance(
        call,
        ast.Call,
    ):

        return None


    for keyword in call.keywords:

        if keyword.arg != "dtype":
            continue

        return call_name(
            keyword.value
        )


    return None


# =============================================================================
# 4. CLEAN INTERRUPTED-STATE GATE
# =============================================================================

banner(
    "STAGE26-4C1-RECOVERY-GIT :: INTERRUPTED STATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)

print(
    "Lock dir exists:",
    LOCK_DIR.exists()
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed after failed 4C1 cell."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed after failed 4C1 cell."
    )


if status:

    raise RuntimeError(
        "Repository changed after failed 4C1 cell."
    )


if LOCK_DIR.exists():

    existing = list(
        LOCK_DIR.iterdir()
    )

    if existing:

        print(
            "\nUnexpected lock files:"
        )

        for path in existing:

            print(
                " ",
                path
            )

        raise RuntimeError(
            "4C1 lock directory unexpectedly contains files."
        )


print(
    "\n[PASS] Previous failure occurred before lock-package creation."
)


# =============================================================================
# 5. UPSTREAM IDENTITY
# =============================================================================

banner(
    "STAGE26-4C1-RECOVERY-GIT :: UPSTREAM IDENTITY"
)


identity_checks = [
    (
        "fresh 4C0 inventory",
        INVENTORY_SOURCE,
        EXPECTED_INVENTORY_SHA256,
    ),

    (
        "Stage20 encoder",
        ENCODER,
        EXPECTED_ENCODER_SHA256,
    ),

    (
        "Stage20 compact loader",
        LOADER,
        EXPECTED_LOADER_SHA256,
    ),

    (
        "Stage20 architecture lock",
        ARCH_LOCK,
        EXPECTED_ARCH_LOCK_SHA256,
    ),

    (
        "Stage26 measurement protocol",
        MEASUREMENT_PROTOCOL,
        EXPECTED_MEASUREMENT_PROTOCOL_SHA256,
    ),

    (
        "Stage26-4B3 raw timing manifest",
        RAW_TIMING_MANIFEST,
        EXPECTED_RAW_TIMING_MANIFEST_SHA256,
    ),
]


for label, path, expected in identity_checks:

    if not path.exists():

        raise FileNotFoundError(
            path
        )


    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:36s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Upstream identity mismatch: {label}"
        )


if not MONDAY_TAR.exists():

    raise FileNotFoundError(
        MONDAY_TAR
    )


if int(
    MONDAY_TAR.stat().st_size
) != EXPECTED_TAR_BYTES:

    raise RuntimeError(
        "Monday Release TAR size changed."
    )


print(
    "\nMonday Release TAR:"
)

print(
    "  size       : PASS"
)

print(
    "  full SHA256:",
    EXPECTED_TAR_SHA256
)

print(
    "  rehash now : NO"
)

print(
    "  extraction : NO"
)


# =============================================================================
# 6. SCIENTIFIC INVENTORY GATE
# =============================================================================

banner(
    "STAGE26-4C1-RECOVERY-GIT :: 4C0 SCIENTIFIC GATE"
)


inventory = json.loads(
    INVENTORY_SOURCE.read_text(
        encoding="utf-8"
    )
)


inventory_checks = {
    "inventory_PASS":
        (
            inventory[
                "status"
            ]
            ==
            "PASS_INVENTORY_REBUILT_AFTER_SESSION_RESET"
        ),

    "parent_exact":
        (
            inventory[
                "scientific_parent"
            ]
            ==
            EXPECTED_PARENT
        ),

    "release_restored":
        (
            inventory[
                "scientific_state"
            ][
                "release_asset_restored"
            ]
            is True
        ),

    "corpus_not_regenerated":
        (
            inventory[
                "scientific_state"
            ][
                "release_corpus_regenerated"
            ]
            is False
        ),

    "representation_not_materialized":
        (
            inventory[
                "scientific_state"
            ][
                "dense_representation_materialized"
            ]
            is False
        ),

    "equivalence_not_run":
        (
            inventory[
                "scientific_state"
            ][
                "representation_equivalence_test_performed"
            ]
            is False
        ),

    "timing_not_run":
        (
            inventory[
                "scientific_state"
            ][
                "representation_timing_performed"
            ]
            is False
        ),

    "pcap_not_accessed":
        (
            inventory[
                "scientific_state"
            ][
                "pcap_accessed"
            ]
            is False
        ),

    "gpu_not_used":
        (
            inventory[
                "scientific_state"
            ][
                "gpu_used"
            ]
            is False
        ),
}


for name, passed in inventory_checks.items():

    print(
        f"{name:40s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    inventory_checks.values()
):

    raise RuntimeError(
        "Fresh 4C0 scientific gate failed."
    )


# =============================================================================
# 7. RE-DERIVE EXACT HISTORICAL OUTPUT CONTRACT
# =============================================================================

banner(
    "STAGE26-4C1-RECOVERY-GIT :: HISTORICAL OUTPUT CONTRACT"
)


loader_text = LOADER.read_text(
    encoding="utf-8"
)

loader_tree = ast.parse(
    loader_text
)


candidates = []


for class_node in loader_tree.body:

    if not isinstance(
        class_node,
        ast.ClassDef,
    ):
        continue


    for method_node in class_node.body:

        if (
            isinstance(
                method_node,
                ast.FunctionDef,
            )
            and
            method_node.name
            ==
            "reconstruct"
        ):

            candidates.append(
                (
                    class_node,
                    method_node,
                )
            )


if len(
    candidates
) != 1:

    raise RuntimeError(
        "Expected exactly one reconstruct() implementation."
    )


loader_class_node, reconstruct_node = (
    candidates[
        0
    ]
)

HISTORICAL_LOADER_CLASS = (
    loader_class_node.name
)


if HISTORICAL_LOADER_CLASS != "Stage20CompactCorpus":

    raise RuntimeError(
        "Unexpected historical loader class."
    )


reconstruct_source = source_segment(
    loader_text,
    reconstruct_node,
)

reconstruct_source_sha = hashlib.sha256(
    reconstruct_source.encode(
        "utf-8"
    )
).hexdigest()


EXPECTED_RECONSTRUCT_SOURCE_SHA256 = (
    "6290b1427ba42a34590ed7847307e7926310d50056afaa55e83713bdf9afce0c"
)


print(
    "Loader class :",
    HISTORICAL_LOADER_CLASS
)

print(
    "Method line  :",
    reconstruct_node.lineno
)

print(
    "Source SHA256:"
)

print(
    " ",
    reconstruct_source_sha
)


if (
    reconstruct_source_sha
    !=
    EXPECTED_RECONSTRUCT_SOURCE_SHA256
):

    raise RuntimeError(
        "Historical reconstruct() source changed."
    )


allocations = []


for node in ast.walk(
    reconstruct_node
):

    if not isinstance(
        node,
        (
            ast.Assign,
            ast.AnnAssign,
        ),
    ):

        continue


    value = getattr(
        node,
        "value",
        None,
    )


    if not isinstance(
        value,
        ast.Call,
    ):

        continue


    function_name = call_name(
        value.func
    )


    if function_name not in {
        "np.zeros",
        "numpy.zeros",
    }:

        continue


    if not value.args:

        continue


    variable = None


    if isinstance(
        node,
        ast.Assign,
    ):

        if (
            len(
                node.targets
            )
            ==
            1
            and
            isinstance(
                node.targets[
                    0
                ],
                ast.Name,
            )
        ):

            variable = node.targets[
                0
            ].id


    elif isinstance(
        node,
        ast.AnnAssign,
    ):

        if isinstance(
            node.target,
            ast.Name,
        ):

            variable = node.target.id


    allocations.append(
        {
            "variable":
                variable,

            "shape":
                resolve_shape(
                    value.args[
                        0
                    ]
                ),

            "dtype":
                resolve_dtype(
                    value
                ),

            "source":
                source_segment(
                    loader_text,
                    node,
                ),
        }
    )


print(
    "\nDerived allocations:"
)


for allocation in allocations:

    print(
        " ",
        allocation
    )


image_matches = [
    row
    for row in allocations
    if (
        row[
            "shape"
        ]
        ==
        [
            64,
            256,
        ]
        and
        row[
            "dtype"
        ]
        in {
            "np.uint8",
            "numpy.uint8",
        }
    )
]


mask_matches = [
    row
    for row in allocations
    if (
        row[
            "shape"
        ]
        ==
        [
            64,
            256,
        ]
        and
        row[
            "dtype"
        ]
        in {
            "np.bool_",
            "numpy.bool_",
            "bool",
        }
    )
]


if len(
    image_matches
) != 1:

    raise RuntimeError(
        "Historical uint8 [64,256] image allocation not unique."
    )


if len(
    mask_matches
) != 1:

    raise RuntimeError(
        "Historical bool [64,256] padding-mask allocation not unique."
    )


IMAGE_VARIABLE = (
    image_matches[
        0
    ][
        "variable"
    ]
)

MASK_VARIABLE = (
    mask_matches[
        0
    ][
        "variable"
    ]
)


# Verify both are actually returned.
returned_names = set()


for node in ast.walk(
    reconstruct_node
):

    if not isinstance(
        node,
        ast.Return,
    ):

        continue


    value = node.value


    if isinstance(
        value,
        (
            ast.Tuple,
            ast.List,
        ),
    ):

        for element in value.elts:

            if isinstance(
                element,
                ast.Name,
            ):

                returned_names.add(
                    element.id
                )


if IMAGE_VARIABLE not in returned_names:

    raise RuntimeError(
        "Historical image variable not returned."
    )


if MASK_VARIABLE not in returned_names:

    raise RuntimeError(
        "Historical padding-mask variable not returned."
    )


print(
    "\n[PASS] Exact historical contract:"
)

print(
    "  image        :",
    IMAGE_VARIABLE,
    "uint8 [64,256]"
)

print(
    "  padding_mask :",
    MASK_VARIABLE,
    "bool [64,256]"
)


# =============================================================================
# 8. CREATE LOCK DIRECTORY
# =============================================================================

LOCK_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


shutil.copy2(
    INVENTORY_SOURCE,
    INVENTORY_COPY,
)


if (
    sha256_file(
        INVENTORY_COPY
    )
    !=
    EXPECTED_INVENTORY_SHA256
):

    raise RuntimeError(
        "Inventory copy changed."
    )


# =============================================================================
# 9. BOUNDARY MAP
# =============================================================================

banner(
    "STAGE26-4C1-RECOVERY-GIT :: FREEZE BOUNDARY"
)


boundary = {
    "schema":
        "stage26_representation_boundary_map_v3",

    "scientific_parent":
        EXPECTED_PARENT,

    "group":
        "GROUP_B_PACKET_IMAGE",

    "component_name":
        "STAGE20_COMPACT_CORPUS_TO_DENSE_UINT8_IMAGE_AND_BYTE_PADDING_MASK",

    "historical_contract": {
        "loader_class":
            HISTORICAL_LOADER_CLASS,

        "reconstruct_source_sha256":
            reconstruct_source_sha,

        "image_variable":
            IMAGE_VARIABLE,

        "image_shape":
            [
                64,
                256,
            ],

        "image_dtype":
            "uint8",

        "padding_mask_variable":
            MASK_VARIABLE,

        "padding_mask_shape":
            [
                64,
                256,
            ],

        "padding_mask_dtype":
            "bool",
    },

    "authoritative_input": {
        "release_tag":
            RELEASE_TAG,

        "release_asset":
            RELEASE_ASSET,

        "release_tar_sha256":
            EXPECTED_TAR_SHA256,

        "population_flow_count":
            FLOW_COUNT,

        "encoded_byte_count":
            ENCODED_BYTE_COUNT,

        "component_input_files": [
            "encoded_bytes.bin",
            "flow_offsets.npy",
            "packet_lengths.npy",
        ],

        "labels_used":
            False,
    },

    "output": {
        "image_shape_per_flow":
            [
                64,
                256,
            ],

        "image_dtype":
            "uint8",

        "padding_mask_shape_per_flow":
            [
                64,
                256,
            ],

        "padding_mask_dtype":
            "bool",

        "batch_image_shape":
            "[B,64,256]",

        "batch_padding_mask_shape":
            "[B,64,256]",
    },

    "included_operations": [
        "read packet_lengths row",
        "read flow byte offsets",
        "allocate zero-filled uint8 batch image arrays",
        "allocate zero-filled bool byte-padding-mask arrays",
        "copy retained encoded packet bytes into each packet-row prefix",
        "set authentic byte positions True in byte-padding mask",
        "validate flow offset delta equals sum(packet_lengths)",
    ],

    "excluded_operations": [
        "GitHub Release download",
        "Release TAR restoration",
        "raw PCAP reading",
        "flow reconstruction",
        "supervised matching/filtering",
        "packet selection",
        "header masking",
        "packet truncation/compact encoding",
        "labels.npy access",
        "float32 conversion",
        "division by 255",
        "torch conversion",
        "CNN inference",
        "ViT inference",
        "thresholding",
    ],

    "model_scaling_boundary": {
        "function":
            "scale_for_model",

        "conversion":
            "np.float32",

        "denominator":
            "np.float32(255.0)",

        "included_in_representation_materialization":
            False,
    },

    "claim_boundary": {
        "compact_corpus_dense_materialization_cost":
            True,

        "raw_flow_to_packet_image_complete_cost":
            False,

        "packet_masking_encoding_cost_included":
            False,

        "group_A_70_feature_extraction_profiled":
            False,

        "complete_end_to_end_claim_allowed":
            False,

        "result_is_sample_specific":
            True,
    },
}


atomic_json(
    BOUNDARY_PATH,
    boundary,
)


boundary_sha = sha256_file(
    BOUNDARY_PATH
)


print(
    "Boundary SHA256:"
)

print(
    " ",
    boundary_sha
)


# =============================================================================
# 10. CORRECTED REPRESENTATION WORKER
# =============================================================================

banner(
    "STAGE26-4C1-RECOVERY-GIT :: FREEZE WORKER"
)


WORKER_TEXT = r'''#!/usr/bin/env python3
from __future__ import annotations

import os
import sys
import json
import time
import hashlib
import importlib.util
from pathlib import Path

import numpy as np


ROWS = 64
COLS = 256


def atomic_json(path: Path, obj) -> None:

    path = Path(path)
    tmp = Path(str(path) + ".tmp")

    with tmp.open("w", encoding="utf-8") as fh:

        json.dump(
            obj,
            fh,
            indent=2,
            sort_keys=True,
        )

        fh.write("\n")
        fh.flush()
        os.fsync(fh.fileno())

    os.replace(tmp, path)


def fingerprint(
    image: np.ndarray,
    padding_mask: np.ndarray,
) -> str:

    h = hashlib.sha256()

    h.update(
        np.ascontiguousarray(
            image
        ).tobytes()
    )

    h.update(
        np.ascontiguousarray(
            padding_mask
        ).tobytes()
    )

    return h.hexdigest()


class RepresentationSource:

    def __init__(self, corpus_dir: Path):

        self.corpus_dir = Path(corpus_dir)

        self.encoded_path = (
            self.corpus_dir
            / "encoded_bytes.bin"
        )

        self.offsets_path = (
            self.corpus_dir
            / "flow_offsets.npy"
        )

        self.lengths_path = (
            self.corpus_dir
            / "packet_lengths.npy"
        )

        for path in (
            self.encoded_path,
            self.offsets_path,
            self.lengths_path,
        ):

            if not path.is_file():

                raise FileNotFoundError(
                    path
                )

        self.flow_offsets = np.load(
            self.offsets_path,
            mmap_mode="r",
            allow_pickle=False,
        )

        self.packet_lengths = np.load(
            self.lengths_path,
            mmap_mode="r",
            allow_pickle=False,
        )

        self.encoded_bytes = np.memmap(
            self.encoded_path,
            dtype=np.uint8,
            mode="r",
        )

        if (
            self.packet_lengths.ndim != 2
            or
            self.packet_lengths.shape[1] != ROWS
        ):

            raise ValueError(
                "packet_lengths.npy must be [N,64]"
            )

        self.n = int(
            self.packet_lengths.shape[0]
        )

        if self.flow_offsets.shape != (
            self.n + 1,
        ):

            raise ValueError(
                "flow_offsets.npy must be [N+1]"
            )

        if self.packet_lengths.dtype != np.uint16:

            raise ValueError(
                "packet_lengths dtype must be uint16"
            )

        if self.flow_offsets.dtype != np.uint64:

            raise ValueError(
                "flow_offsets dtype must be uint64"
            )

        if int(
            self.flow_offsets[0]
        ) != 0:

            raise ValueError(
                "first offset must be zero"
            )

        if int(
            self.flow_offsets[-1]
        ) != int(
            self.encoded_bytes.size
        ):

            raise ValueError(
                "final offset != encoded byte count"
            )


    def reconstruct(
        self,
        index: int,
    ) -> tuple[np.ndarray, np.ndarray]:

        index = int(index)

        if index < 0:
            index += self.n

        if index < 0 or index >= self.n:

            raise IndexError(
                index
            )

        lengths = np.asarray(
            self.packet_lengths[index],
            dtype=np.uint16,
        )

        if np.any(
            lengths > COLS
        ):

            raise ValueError(
                "packet length exceeds 256"
            )

        # Frozen geometry: retained packet rows are contiguous from row 0.
        zero_seen = False

        for value in lengths.tolist():

            value = int(value)

            if value == 0:

                zero_seen = True

            elif zero_seen:

                raise ValueError(
                    "positive packet length after zero-padding row"
                )

        start = int(
            self.flow_offsets[index]
        )

        end = int(
            self.flow_offsets[index + 1]
        )

        expected = int(
            lengths.astype(
                np.uint64
            ).sum()
        )

        if end - start != expected:

            raise ValueError(
                "offset delta != sum(packet_lengths)"
            )

        image = np.zeros(
            (
                ROWS,
                COLS,
            ),
            dtype=np.uint8,
        )

        padding_mask = np.zeros(
            (
                ROWS,
                COLS,
            ),
            dtype=np.bool_,
        )

        cursor = start

        for row_index, packet_length in enumerate(
            lengths.tolist()
        ):

            packet_length = int(
                packet_length
            )

            if packet_length == 0:
                continue

            next_cursor = (
                cursor
                +
                packet_length
            )

            image[
                row_index,
                :packet_length,
            ] = self.encoded_bytes[
                cursor:
                next_cursor
            ]

            padding_mask[
                row_index,
                :packet_length,
            ] = True

            cursor = next_cursor

        if cursor != end:

            raise ValueError(
                "byte traversal ended at unexpected offset"
            )

        return (
            image,
            padding_mask,
        )


    def materialize_batch(
        self,
        indices: np.ndarray,
    ) -> tuple[np.ndarray, np.ndarray]:

        indices = np.asarray(
            indices,
            dtype=np.int64,
        ).reshape(-1)

        batch_size = int(
            indices.size
        )

        images = np.zeros(
            (
                batch_size,
                ROWS,
                COLS,
            ),
            dtype=np.uint8,
        )

        padding_masks = np.zeros(
            (
                batch_size,
                ROWS,
                COLS,
            ),
            dtype=np.bool_,
        )

        for batch_index, flow_index in enumerate(
            indices.tolist()
        ):

            image, padding_mask = self.reconstruct(
                int(
                    flow_index
                )
            )

            images[
                batch_index
            ] = image

            padding_masks[
                batch_index
            ] = padding_mask

        return (
            images,
            padding_masks,
        )


def import_module(
    name: str,
    path: Path,
):

    spec = importlib.util.spec_from_file_location(
        name,
        path,
    )

    if spec is None or spec.loader is None:

        raise RuntimeError(
            f"unable to import {path}"
        )

    module = importlib.util.module_from_spec(
        spec
    )

    spec.loader.exec_module(
        module
    )

    return module


def run_equivalence(
    cfg: dict,
    source: RepresentationSource,
) -> dict:

    historical_module = import_module(
        "stage26_historical_compact_loader",
        Path(
            cfg[
                "historical_loader_path"
            ]
        ),
    )

    historical_class = getattr(
        historical_module,
        cfg[
            "historical_loader_class"
        ],
    )

    historical = historical_class(
        Path(
            cfg[
                "corpus_dir"
            ]
        )
    )

    start_index = int(
        cfg[
            "start_index"
        ]
    )

    flow_count = int(
        cfg[
            "flow_count"
        ]
    )

    aggregate = hashlib.sha256()

    for flow_index in range(
        start_index,
        start_index + flow_count,
    ):

        new_image, new_mask = (
            source.reconstruct(
                flow_index
            )
        )

        historical_result = historical.reconstruct(
            flow_index
        )

        if not isinstance(
            historical_result,
            tuple,
        ):

            raise RuntimeError(
                "historical reconstruct() did not return tuple"
            )

        if len(
            historical_result
        ) < 2:

            raise RuntimeError(
                "historical reconstruct() returned fewer than 2 objects"
            )

        old_image = np.asarray(
            historical_result[
                0
            ]
        )

        old_mask = np.asarray(
            historical_result[
                1
            ]
        )

        if old_image.shape != (
            ROWS,
            COLS,
        ):

            raise RuntimeError(
                f"historical image shape mismatch at {flow_index}"
            )

        if old_mask.shape != (
            ROWS,
            COLS,
        ):

            raise RuntimeError(
                f"historical padding-mask shape mismatch at {flow_index}"
            )

        if old_image.dtype != np.uint8:

            raise RuntimeError(
                f"historical image dtype mismatch at {flow_index}"
            )

        if old_mask.dtype != np.bool_:

            raise RuntimeError(
                f"historical mask dtype mismatch at {flow_index}"
            )

        if not np.array_equal(
            new_image,
            old_image,
        ):

            raise RuntimeError(
                f"image equivalence failure at flow {flow_index}"
            )

        if not np.array_equal(
            new_mask,
            old_mask,
        ):

            raise RuntimeError(
                f"padding-mask equivalence failure at flow {flow_index}"
            )

        aggregate.update(
            np.ascontiguousarray(
                new_image
            ).tobytes()
        )

        aggregate.update(
            np.ascontiguousarray(
                new_mask
            ).tobytes()
        )

    return {
        "status":
            "PASS",

        "mode":
            "VALIDATE_EQUIVALENCE",

        "start_index":
            start_index,

        "flow_count":
            flow_count,

        "end_index_exclusive":
            start_index + flow_count,

        "representation_fingerprint_sha256":
            aggregate.hexdigest(),

        "timing_performed":
            False,

        "gpu_used":
            False,
    }


def run_benchmark(
    cfg: dict,
    source: RepresentationSource,
) -> dict:

    start_index = int(
        cfg[
            "start_index"
        ]
    )

    batch_size = int(
        cfg[
            "batch_size"
        ]
    )

    warmup_runs = int(
        cfg[
            "warmup_runs"
        ]
    )

    timed_runs = int(
        cfg[
            "timed_runs"
        ]
    )

    indices = np.arange(
        start_index,
        start_index + batch_size,
        dtype=np.int64,
    )

    warm_fingerprint = None

    for _ in range(
        warmup_runs
    ):

        images, masks = (
            source.materialize_batch(
                indices
            )
        )

        # Outside any timer.
        warm_fingerprint = fingerprint(
            images,
            masks,
        )

    observations = []

    reference_fingerprint = None

    for iteration in range(
        1,
        timed_runs + 1,
    ):

        start_ns = time.perf_counter_ns()

        images, masks = (
            source.materialize_batch(
                indices
            )
        )

        end_ns = time.perf_counter_ns()

        elapsed_ns = int(
            end_ns
            -
            start_ns
        )

        if elapsed_ns <= 0:

            raise RuntimeError(
                "non-positive elapsed time"
            )

        # Integrity check intentionally outside timer.
        output_fingerprint = fingerprint(
            images,
            masks,
        )

        if reference_fingerprint is None:

            reference_fingerprint = (
                output_fingerprint
            )

        elif (
            output_fingerprint
            !=
            reference_fingerprint
        ):

            raise RuntimeError(
                "representation output changed across timed iterations"
            )

        elapsed_seconds = (
            elapsed_ns
            /
            1_000_000_000.0
        )

        observations.append(
            {
                "iteration_index":
                    iteration,

                "elapsed_ns":
                    elapsed_ns,

                "elapsed_seconds":
                    elapsed_seconds,

                "flows":
                    batch_size,

                "flows_per_second":
                    (
                        batch_size
                        /
                        elapsed_seconds
                    ),

                "image_output_bytes":
                    int(
                        images.nbytes
                    ),

                "padding_mask_output_bytes":
                    int(
                        masks.nbytes
                    ),

                "total_output_bytes":
                    int(
                        images.nbytes
                        +
                        masks.nbytes
                    ),
            }
        )

    return {
        "status":
            "PASS",

        "mode":
            "BENCHMARK_TIMED",

        "start_index":
            start_index,

        "batch_size":
            batch_size,

        "end_index_exclusive":
            start_index + batch_size,

        "warmup_runs":
            warmup_runs,

        "timed_runs":
            timed_runs,

        "representation_fingerprint_sha256":
            reference_fingerprint,

        "warm_representation_fingerprint_sha256":
            warm_fingerprint,

        "observations":
            observations,

        "timing_performed":
            True,

        "float32_div255_performed":
            False,

        "model_loaded":
            False,

        "gpu_used":
            False,
    }


def main():

    if len(
        sys.argv
    ) != 2:

        raise SystemExit(
            "usage: stage26_representation_worker_v1.py CONFIG.json"
        )

    config_path = Path(
        sys.argv[
            1
        ]
    )

    cfg = json.loads(
        config_path.read_text(
            encoding="utf-8"
        )
    )

    affinity = cfg.get(
        "affinity"
    )

    if affinity is not None:

        if not hasattr(
            os,
            "sched_setaffinity",
        ):

            raise RuntimeError(
                "sched_setaffinity unavailable"
            )

        os.sched_setaffinity(
            0,
            {
                int(cpu)
                for cpu in affinity
            },
        )

    source = RepresentationSource(
        Path(
            cfg[
                "corpus_dir"
            ]
        )
    )

    if source.n != int(
        cfg[
            "expected_flow_count"
        ]
    ):

        raise RuntimeError(
            "corpus flow count mismatch"
        )

    mode = cfg[
        "mode"
    ]

    if mode == "VALIDATE_EQUIVALENCE":

        result = run_equivalence(
            cfg,
            source,
        )

    elif mode == "BENCHMARK_TIMED":

        result = run_benchmark(
            cfg,
            source,
        )

    else:

        raise RuntimeError(
            f"unknown mode: {mode}"
        )

    result.update(
        {
            "schema":
                "stage26_representation_worker_result_v3",

            "worker_pid":
                os.getpid(),

            "population_flow_count":
                source.n,

            "image_shape_per_flow":
                [
                    ROWS,
                    COLS,
                ],

            "image_dtype":
                "uint8",

            "padding_mask_shape_per_flow":
                [
                    ROWS,
                    COLS,
                ],

            "padding_mask_dtype":
                "bool",

            "corpus_created_or_modified":
                False,

            "pcap_accessed":
                False,

            "labels_accessed":
                False,
        }
    )

    atomic_json(
        Path(
            cfg[
                "result_path"
            ]
        ),
        result,
    )


if __name__ == "__main__":
    main()
'''


# Syntax-check without creating __pycache__.
compile(
    WORKER_TEXT,
    str(
        WORKER_PATH
    ),
    "exec",
)


atomic_text(
    WORKER_PATH,
    WORKER_TEXT,
)


worker_sha = sha256_file(
    WORKER_PATH
)


print(
    "Worker syntax : PASS"
)

print(
    "Worker SHA256:"
)

print(
    " ",
    worker_sha
)


# Static forbidden-import/input audit.
worker_tree = ast.parse(
    WORKER_TEXT
)


imports = set()


for node in ast.walk(
    worker_tree
):

    if isinstance(
        node,
        ast.Import,
    ):

        for alias in node.names:

            imports.add(
                alias.name.split(
                    "."
                )[
                    0
                ]
            )


    elif isinstance(
        node,
        ast.ImportFrom,
    ):

        if node.module:

            imports.add(
                node.module.split(
                    "."
                )[
                    0
                ]
            )


for forbidden in {
    "scapy",
    "dpkt",
    "pyshark",
    "torch",
}:

    if forbidden in imports:

        raise RuntimeError(
            f"Forbidden worker import: {forbidden}"
        )


if (
    '"labels.npy"' in WORKER_TEXT
    or
    "'labels.npy'" in WORKER_TEXT
):

    raise RuntimeError(
        "Worker unexpectedly references labels.npy."
    )


print(
    "Worker forbidden-source audit: PASS"
)


# =============================================================================
# 11. FREEZE CORRECTED REPRESENTATION PROTOCOL
# =============================================================================

banner(
    "STAGE26-4C1-RECOVERY-GIT :: FREEZE PROTOCOL"
)


protocol = {
    "schema":
        "stage26_representation_protocol_v3",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4C1",

    "status":
        "FROZEN_BEFORE_FIRST_REPRESENTATION_MEASUREMENT",

    "scientific_parent":
        EXPECTED_PARENT,

    "selection_independence": {
        "sample_selected_using_inference_results":
            False,

        "sample_selected_using_representation_timing":
            False,

        "sample_selected_using_labels":
            False,

        "seed":
            SEED,

        "selection_rule":
            (
                "Existing Stage26 seed 26042 modulo valid start "
                "positions for the largest frozen batch."
            ),
    },

    "authoritative_source": {
        "release_tag":
            RELEASE_TAG,

        "release_asset":
            RELEASE_ASSET,

        "release_tar_bytes":
            EXPECTED_TAR_BYTES,

        "release_tar_sha256":
            EXPECTED_TAR_SHA256,

        "population_flow_count":
            FLOW_COUNT,

        "member_identities":
            MEMBER_IDENTITIES,

        "restore_before_equivalence":
            True,

        "restore_operation_timed":
            False,

        "restore_rule":
            (
                "Restore exact already-published Release members "
                "byte-for-byte to isolated runtime storage and verify "
                "every required member SHA256. Restoration is not "
                "corpus regeneration."
            ),
    },

    "implementation": {
        "fresh_4c0_inventory_path":
            str(
                INVENTORY_COPY.relative_to(
                    REPO
                )
            ),

        "fresh_4c0_inventory_sha256":
            EXPECTED_INVENTORY_SHA256,

        "historical_encoder_path":
            str(
                ENCODER.relative_to(
                    REPO
                )
            ),

        "historical_encoder_sha256":
            EXPECTED_ENCODER_SHA256,

        "historical_loader_path":
            str(
                LOADER.relative_to(
                    REPO
                )
            ),

        "historical_loader_sha256":
            EXPECTED_LOADER_SHA256,

        "historical_loader_class":
            HISTORICAL_LOADER_CLASS,

        "historical_reconstruct_source_sha256":
            reconstruct_source_sha,

        "worker_path":
            str(
                WORKER_PATH.relative_to(
                    REPO
                )
            ),

        "worker_sha256":
            worker_sha,

        "boundary_map_path":
            str(
                BOUNDARY_PATH.relative_to(
                    REPO
                )
            ),

        "boundary_map_sha256":
            boundary_sha,
    },

    "historical_output_contract": {
        "image_shape":
            [
                64,
                256,
            ],

        "image_dtype":
            "uint8",

        "padding_mask_shape":
            [
                64,
                256,
            ],

        "padding_mask_dtype":
            "bool",

        "padding_mask_semantics":
            (
                "Byte-level authenticity mask: True for retained "
                "encoded byte positions and False for zero-padding."
            ),
    },

    "mandatory_pre_timing_equivalence_gate": {
        "required":
            True,

        "start_index_zero_based":
            EQUIVALENCE_START,

        "flow_count":
            EQUIVALENCE_FLOW_COUNT,

        "comparison":
            (
                "For each gated flow, new worker image and byte-level "
                "padding_mask must be exactly np.array_equal to the "
                "first two outputs of frozen "
                "Stage20CompactCorpus.reconstruct()."
            ),

        "timing_during_equivalence":
            False,

        "timing_allowed_before_gate_pass":
            False,

        "gate_result_must_be_git_anchored_before_timing":
            True,
    },

    "measurement_boundary": {
        "input":
            "RESTORED_EXACT_STAGE20_COMPACT_RELEASE_FILES",

        "output":
            "UINT8_IMAGE_BATCH_AND_BOOL_BYTE_PADDING_MASK_BATCH",

        "batch_image_shape":
            "[B,64,256]",

        "batch_padding_mask_shape":
            "[B,64,256]",

        "batch_output_allocation_included":
            True,

        "memmap_reads_included":
            True,

        "release_download_included":
            False,

        "release_tar_restoration_included":
            False,

        "loader_initialization_included":
            False,

        "Python_process_startup_included":
            False,

        "JSON_serialization_included":
            False,

        "fingerprinting_included":
            False,

        "labels_accessed":
            False,

        "float32_conversion_included":
            False,

        "division_by_255_included":
            False,

        "model_forward_included":
            False,

        "page_cache_manipulated":
            False,

        "warm_exact_batch_before_timing":
            True,

        "cold_disk_IO_claim_allowed":
            False,

        "interpretation":
            (
                "Warm compact-corpus dense-materialization cost "
                "under existing OS page-cache state."
            ),
    },

    "sample": {
        "population_flow_count":
            FLOW_COUNT,

        "sampling_rule":
            "DETERMINISTIC_NESTED_CONTIGUOUS_SEGMENT",

        "start_index_zero_based":
            SAMPLE_START,

        "largest_end_index_zero_based_exclusive":
            SAMPLE_END_EXCLUSIVE,

        "largest_batch_flow_count":
            MAX_BATCH_SIZE,

        "largest_batch_population_fraction":
            (
                MAX_BATCH_SIZE
                /
                FLOW_COUNT
            ),

        "batch_sizes":
            BATCH_SIZES,

        "nested_rule":
            (
                "Batch B uses flows [start,start+B); "
                "all smaller conditions are prefixes of the largest."
            ),

        "sample_specific_claim_required":
            True,
    },

    "hardware": {
        "mode":
            "CPU_1_PHYSICAL_CORE",

        "affinity":
            CPU_AFFINITY,

        "thread_count":
            THREAD_COUNT,

        "gpu":
            False,
    },

    "iteration_policy": {
        str(
            batch
        ):
            ITERATION_POLICY[
                batch
            ]
        for batch in BATCH_SIZES
    },

    "condition_count":
        len(
            BATCH_SIZES
        ),

    "fresh_process_per_batch_condition":
        True,

    "condition_timeout_seconds":
        CONDITION_TIMEOUT_SECONDS,

    "environment_gate":
        ENVIRONMENT_GATE,

    "failure_policy": {
        "INVALID_ENVIRONMENT":
            "Do not benchmark condition.",

        "TIMEOUT_RESOURCE_LIMIT":
            "Record timeout; do not adapt.",

        "RESOURCE_LIMIT_OOM":
            "Record resource outcome; do not reduce batch.",

        "IMPLEMENTATION_FAILURE":
            (
                "Stop for narrow implementation diagnosis; "
                "do not alter frozen scientific protocol."
            ),

        "no_post_result_adaptation":
            True,
    },

    "required_raw_metrics": [
        "batch_size",
        "iteration_index",
        "elapsed_ns",
        "elapsed_seconds",
        "flows_per_second",
        "image_output_bytes",
        "padding_mask_output_bytes",
        "total_output_bytes",
    ],

    "summary_metrics": [
        "p50_batch_latency",
        "p95_batch_latency",
        "p99_batch_latency_when_n_gte_100",
        "median_flows_per_second",
        "minimum_flows_per_second",
        "maximum_flows_per_second",
    ],

    "claim_boundary": {
        "component_name":
            (
                "Stage20 compact Release corpus -> dense uint8 image "
                "+ bool byte-level padding mask"
            ),

        "applies_to":
            "GROUP_B_PACKET_IMAGE",

        "applies_to_group_A_70_feature_models":
            False,

        "raw_flow_to_packet_image_complete":
            False,

        "packet_masking_encoding_cost_included":
            False,

        "complete_pipeline_additivity_claim_allowed":
            False,

        "representation_results_sample_specific":
            True,
    },

    "scientific_rules": {
        "release_corpus_authoritative":
            True,

        "release_corpus_regeneration_forbidden":
            True,

        "pcap_may_be_opened":
            False,

        "labels_may_be_opened":
            False,

        "models_may_be_loaded":
            False,

        "Thursday_may_be_accessed":
            False,

        "Friday_may_be_accessed":
            False,

        "GPU_used":
            False,

        "first_representation_timing_allowed_only_after":
            (
                "this protocol is remotely Git-anchored; exact Release "
                "members are restored and SHA-verified; the frozen "
                "128-flow untimed equivalence gate passes; and the "
                "equivalence result is remotely Git-anchored."
            ),
    },
}


atomic_json(
    PROTOCOL_PATH,
    protocol,
)


protocol_sha = sha256_file(
    PROTOCOL_PATH
)


print(
    "Protocol SHA256:"
)

print(
    " ",
    protocol_sha
)


# =============================================================================
# 12. UPSTREAM PROVENANCE
# =============================================================================

provenance = {
    "schema":
        "stage26_4c_upstream_provenance_v3",

    "scientific_parent":
        EXPECTED_PARENT,

    "fresh_4c0_inventory": {
        "repo_relative_path":
            str(
                INVENTORY_COPY.relative_to(
                    REPO
                )
            ),

        "sha256":
            EXPECTED_INVENTORY_SHA256,
    },

    "stage26_raw_extraction": {
        "manifest_repo_relative_path":
            str(
                RAW_TIMING_MANIFEST.relative_to(
                    REPO
                )
            ),

        "manifest_sha256":
            EXPECTED_RAW_TIMING_MANIFEST_SHA256,

        "closed":
            True,
    },

    "stage20": {
        "encoder": {
            "repo_relative_path":
                str(
                    ENCODER.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_ENCODER_SHA256,
        },

        "compact_loader": {
            "repo_relative_path":
                str(
                    LOADER.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_LOADER_SHA256,

            "class":
                HISTORICAL_LOADER_CLASS,

            "reconstruct_source_sha256":
                reconstruct_source_sha,

            "derived_output_contract": {
                "image_shape":
                    [
                        64,
                        256,
                    ],

                "image_dtype":
                    "uint8",

                "padding_mask_shape":
                    [
                        64,
                        256,
                    ],

                "padding_mask_dtype":
                    "bool",
            },
        },

        "architecture_lock": {
            "repo_relative_path":
                str(
                    ARCH_LOCK.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_ARCH_LOCK_SHA256,
        },
    },

    "stage26_measurement_protocol": {
        "repo_relative_path":
            str(
                MEASUREMENT_PROTOCOL.relative_to(
                    REPO
                )
            ),

        "sha256":
            EXPECTED_MEASUREMENT_PROTOCOL_SHA256,
    },

    "release": {
        "tag":
            RELEASE_TAG,

        "asset":
            RELEASE_ASSET,

        "tar_bytes":
            EXPECTED_TAR_BYTES,

        "tar_sha256":
            EXPECTED_TAR_SHA256,

        "members":
            MEMBER_IDENTITIES,
    },

    "geometry": {
        "flow_count":
            FLOW_COUNT,

        "rows":
            64,

        "columns":
            256,

        "encoded_byte_count":
            ENCODED_BYTE_COUNT,
    },
}


atomic_json(
    PROVENANCE_PATH,
    provenance,
)


provenance_sha = sha256_file(
    PROVENANCE_PATH
)


# =============================================================================
# 13. FREEZE RECEIPT
# =============================================================================

freeze_receipt = {
    "schema":
        "stage26_4c1_representation_protocol_freeze_receipt_v3",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4C1",

    "status":
        "REPRESENTATION_PROTOCOL_FROZEN_NOT_YET_EXECUTED",

    "scientific_parent":
        EXPECTED_PARENT,

    "fresh_4c0_inventory_sha256":
        EXPECTED_INVENTORY_SHA256,

    "worker_sha256":
        worker_sha,

    "protocol_sha256":
        protocol_sha,

    "boundary_map_sha256":
        boundary_sha,

    "upstream_provenance_sha256":
        provenance_sha,

    "historical_contract": {
        "loader_class":
            HISTORICAL_LOADER_CLASS,

        "reconstruct_source_sha256":
            reconstruct_source_sha,

        "image_shape":
            [
                64,
                256,
            ],

        "image_dtype":
            "uint8",

        "padding_mask_shape":
            [
                64,
                256,
            ],

        "padding_mask_dtype":
            "bool",
    },

    "frozen_condition": {
        "CPU":
            "CPU_1_PHYSICAL_CORE",

        "affinity":
            CPU_AFFINITY,

        "thread_count":
            THREAD_COUNT,

        "batch_sizes":
            BATCH_SIZES,

        "sample_start_index_zero_based":
            SAMPLE_START,

        "largest_sample_end_index_exclusive":
            SAMPLE_END_EXCLUSIVE,

        "seed":
            SEED,
    },

    "mandatory_pre_timing_gate": {
        "equivalence_required":
            True,

        "flow_count":
            EQUIVALENCE_FLOW_COUNT,

        "start_index_zero_based":
            EQUIVALENCE_START,

        "must_be_git_anchored_before_timing":
            True,
    },

    "scientific_state": {
        "release_tar_extracted":
            False,

        "dense_representation_materialized":
            False,

        "equivalence_executed":
            False,

        "representation_timing_performed":
            False,

        "pcap_accessed":
            False,

        "release_corpus_regenerated":
            False,

        "labels_accessed":
            False,

        "models_loaded":
            False,

        "inference_performed":
            False,

        "Thursday_accessed":
            False,

        "Friday_accessed":
            False,

        "gpu_used":
            False,
    },
}


atomic_json(
    FREEZE_RECEIPT_PATH,
    freeze_receipt,
)


freeze_receipt_sha = sha256_file(
    FREEZE_RECEIPT_PATH
)


# =============================================================================
# 14. BUILD LOCK MANIFEST
# =============================================================================

banner(
    "STAGE26-4C1-RECOVERY-GIT :: BUILD MANIFEST"
)


package_files = [
    INVENTORY_COPY,
    WORKER_PATH,
    PROTOCOL_PATH,
    BOUNDARY_PATH,
    PROVENANCE_PATH,
    FREEZE_RECEIPT_PATH,
]


manifest_rows = []


for path in package_files:

    manifest_rows.append(
        {
            "repo_relative_path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest = {
    "schema":
        "stage26_4c_representation_lock_manifest_v3",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        "READY_FOR_GIT_ANCHOR",

    "parent_commit":
        EXPECTED_PARENT,

    "file_count_excluding_manifest":
        len(
            manifest_rows
        ),

    "fresh_4c0_inventory_sha256":
        EXPECTED_INVENTORY_SHA256,

    "historical_reconstruct_source_sha256":
        reconstruct_source_sha,

    "worker_sha256":
        worker_sha,

    "protocol_sha256":
        protocol_sha,

    "boundary_sha256":
        boundary_sha,

    "provenance_sha256":
        provenance_sha,

    "freeze_receipt_sha256":
        freeze_receipt_sha,

    "files":
        manifest_rows,
}


atomic_json(
    MANIFEST_PATH,
    manifest,
)


manifest_sha = sha256_file(
    MANIFEST_PATH
)


print(
    "Inventory  :",
    EXPECTED_INVENTORY_SHA256
)

print(
    "Reconstruct:",
    reconstruct_source_sha
)

print(
    "Worker     :",
    worker_sha
)

print(
    "Protocol   :",
    protocol_sha
)

print(
    "Boundary   :",
    boundary_sha
)

print(
    "Provenance :",
    provenance_sha
)

print(
    "Receipt    :",
    freeze_receipt_sha
)

print(
    "Manifest   :",
    manifest_sha
)


# =============================================================================
# 15. LOCAL PACKAGE AUDIT
# =============================================================================

banner(
    "STAGE26-4C1-RECOVERY-GIT :: LOCAL PACKAGE AUDIT"
)


for row in manifest_rows:

    path = (
        REPO
        / row[
            "repo_relative_path"
        ]
    )

    actual_size = int(
        path.stat().st_size
    )

    actual_sha = sha256_file(
        path
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Local package audit failed."
        )


# =============================================================================
# 16. REPOSITORY CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-4C1-RECOVERY-GIT :: REPOSITORY CHANGE AUDIT"
)


repo_status = git(
    "status",
    "--porcelain",
)


print(
    repo_status
)


if not repo_status:

    raise RuntimeError(
        "Expected uncommitted 4C1 lock package."
    )


unexpected = []


for line in repo_status.splitlines():

    rel = line[
        3:
    ]


    if not rel.startswith(
        str(
            LOCK_REL
        )
        +
        "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected Git changes:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 17. GIT IDENTITY
# =============================================================================

author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_PARENT,
)

author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_PARENT,
)


if (
    not author_name.strip()
    or
    "@"
    not in
    author_email
):

    raise RuntimeError(
        "Could not recover valid Git identity."
    )


git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


print(
    "\nGit author:"
)

print(
    " ",
    author_name,
    "<" + author_email + ">"
)


# =============================================================================
# 18. COMMIT
# =============================================================================

banner(
    "STAGE26-4C1-RECOVERY-GIT :: COMMIT"
)


git(
    "add",
    str(
        LOCK_REL
    ),
)


staged = git(
    "diff",
    "--cached",
    "--name-status",
)


print(
    staged
)


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Commit parent mismatch."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository not clean after commit."
    )


# =============================================================================
# 19. PUSH
# =============================================================================

banner(
    "STAGE26-4C1-RECOVERY-GIT :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle secret GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    pushed = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
        check=True,
        text=True,
    )


    print(
        pushed.stdout.strip()
    )


github_token = None


# =============================================================================
# 20. REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-4C1-RECOVERY-GIT :: REMOTE COMMIT"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "origin/main does not equal 4C1 commit."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote subject mismatch."
    )


# =============================================================================
# 21. REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-4C1-RECOVERY-GIT :: REMOTE BYTE VERIFICATION"
)


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    str(
        MANIFEST_PATH.relative_to(
            REPO
        )
    ),
)

remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "Remote manifest SHA256:"
)

print(
    " ",
    remote_manifest_sha
)


if remote_manifest_sha != manifest_sha:

    raise RuntimeError(
        "Remote manifest mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


if (
    remote_manifest[
        "file_count_excluding_manifest"
    ]
    !=
    6
):

    raise RuntimeError(
        "Unexpected remote package file count."
    )


for row in remote_manifest[
    "files"
]:

    data = git_blob_bytes(
        "origin/main",
        row[
            "repo_relative_path"
        ],
    )

    actual_size = len(
        data
    )

    actual_sha = sha256_bytes(
        data
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Remote byte verification failed."
        )


# =============================================================================
# 22. REMOTE SCIENTIFIC AUDIT
# =============================================================================

banner(
    "STAGE26-4C1-RECOVERY-GIT :: REMOTE SCIENTIFIC AUDIT"
)


remote_protocol = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            PROTOCOL_PATH.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)


remote_receipt = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            FREEZE_RECEIPT_PATH.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)


remote_checks = {
    "protocol_frozen":
        (
            remote_protocol[
                "status"
            ]
            ==
            "FROZEN_BEFORE_FIRST_REPRESENTATION_MEASUREMENT"
        ),

    "inventory_exact":
        (
            remote_protocol[
                "implementation"
            ][
                "fresh_4c0_inventory_sha256"
            ]
            ==
            EXPECTED_INVENTORY_SHA256
        ),

    "reconstruct_exact":
        (
            remote_protocol[
                "implementation"
            ][
                "historical_reconstruct_source_sha256"
            ]
            ==
            EXPECTED_RECONSTRUCT_SOURCE_SHA256
        ),

    "image_contract":
        (
            remote_protocol[
                "historical_output_contract"
            ][
                "image_shape"
            ]
            ==
            [
                64,
                256,
            ]
        ),

    "byte_mask_contract":
        (
            remote_protocol[
                "historical_output_contract"
            ][
                "padding_mask_shape"
            ]
            ==
            [
                64,
                256,
            ]
        ),

    "equivalence_required":
        (
            remote_protocol[
                "mandatory_pre_timing_equivalence_gate"
            ][
                "required"
            ]
            is True
        ),

    "equivalence_untimed":
        (
            remote_protocol[
                "mandatory_pre_timing_equivalence_gate"
            ][
                "timing_during_equivalence"
            ]
            is False
        ),

    "equivalence_anchor_required":
        (
            remote_protocol[
                "mandatory_pre_timing_equivalence_gate"
            ][
                "gate_result_must_be_git_anchored_before_timing"
            ]
            is True
        ),

    "div255_excluded":
        (
            remote_protocol[
                "measurement_boundary"
            ][
                "division_by_255_included"
            ]
            is False
        ),

    "labels_excluded":
        (
            remote_protocol[
                "measurement_boundary"
            ][
                "labels_accessed"
            ]
            is False
        ),

    "E2E_blocked":
        (
            remote_protocol[
                "claim_boundary"
            ][
                "complete_pipeline_additivity_claim_allowed"
            ]
            is False
        ),

    "receipt_not_executed":
        (
            remote_receipt[
                "status"
            ]
            ==
            "REPRESENTATION_PROTOCOL_FROZEN_NOT_YET_EXECUTED"
        ),

    "equivalence_false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "equivalence_executed"
            ]
            is False
        ),

    "timing_false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "representation_timing_performed"
            ]
            is False
        ),

    "corpus_not_regenerated":
        (
            remote_receipt[
                "scientific_state"
            ][
                "release_corpus_regenerated"
            ]
            is False
        ),

    "gpu_false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "gpu_used"
            ]
            is False
        ),
}


for name, passed in remote_checks.items():

    print(
        f"{name:42s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    remote_checks.values()
):

    raise RuntimeError(
        "Remote scientific audit failed."
    )


# =============================================================================
# 23. FINAL AUDIT
# =============================================================================

banner(
    "STAGE26-4C1-RECOVERY-GIT :: FINAL AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != final_remote:

    raise RuntimeError(
        "Local/remote divergence."
    )


if final_status:

    raise RuntimeError(
        "Repository not clean."
    )


# =============================================================================
# 24. CLOSURE
# =============================================================================

banner(
    "STAGE26-4C1 RECOVERY + REMOTE ANCHOR COMPLETE"
)


print(
    "NEW DURABLE COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nPARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nHISTORICAL CONTRACT:"
)

print(
    "  loader        :",
    HISTORICAL_LOADER_CLASS
)

print(
    "  reconstruct   :",
    EXPECTED_RECONSTRUCT_SOURCE_SHA256
)

print(
    "  image         : uint8 [64,256]"
)

print(
    "  padding mask  : bool  [64,256]"
)

print(
    "  mask semantics: byte-level authentic-vs-padding positions"
)


print(
    "\nFROZEN SAMPLE:"
)

print(
    "  seed        :",
    SEED
)

print(
    "  start       :",
    SAMPLE_START
)

print(
    "  largest end :",
    SAMPLE_END_EXCLUSIVE
)

print(
    "  batches     :",
    BATCH_SIZES
)


print(
    "\nLOCK HASHES:"
)

print(
    "  inventory  :",
    EXPECTED_INVENTORY_SHA256
)

print(
    "  worker     :",
    worker_sha
)

print(
    "  protocol   :",
    protocol_sha
)

print(
    "  boundary   :",
    boundary_sha
)

print(
    "  provenance :",
    provenance_sha
)

print(
    "  receipt    :",
    freeze_receipt_sha
)

print(
    "  manifest   :",
    manifest_sha
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  local == origin/main  : PASS"
)

print(
    "  parent                : PASS"
)

print(
    "  six package files     : PASS"
)

print(
    "  manifest              : PASS"
)

print(
    "  scientific content    : PASS"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  4C1 PROTOCOL REMOTELY LOCKED    : YES"
)

print(
    "  RELEASE TAR EXTRACTED           : NO"
)

print(
    "  REPRESENTATION MATERIALIZED     : NO"
)

print(
    "  EQUIVALENCE EXECUTED            : NO"
)

print(
    "  REPRESENTATION TIMING           : NO"
)

print(
    "  PCAP ACCESSED                   : NO"
)

print(
    "  CORPUS REGENERATED              : NO"
)

print(
    "  LABELS ACCESSED                 : NO"
)

print(
    "  MODELS LOADED                   : NO"
)

print(
    "  COMPLETE E2E CLAIM              : NO"
)

print(
    "  GPU                             : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Restore ONLY the three required existing Release members"
)

print(
    "  (encoded_bytes.bin, flow_offsets.npy, packet_lengths.npy)"
)

print(
    "  into isolated runtime storage, verify their exact hashes,"
)

print(
    "  then execute the frozen 128-flow UNTIMED equivalence gate."
)

print(
    "  Timing remains forbidden until that equivalence result is"
)

print(
    "  separately committed and pushed."
)


STAGE26-4C1-RECOVERY-GIT :: INTERRUPTED STATE
Expected parent: 55aba2e1cc08385659479c6275234dc23b11d231
Local HEAD     : 55aba2e1cc08385659479c6275234dc23b11d231
origin/main    : 55aba2e1cc08385659479c6275234dc23b11d231
Repo clean     : True
Lock dir exists: False

[PASS] Previous failure occurred before lock-package creation.

STAGE26-4C1-RECOVERY-GIT :: UPSTREAM IDENTITY
fresh 4C0 inventory                  PASS b32a2a21407261d0c3628e80d1c39aab1b6921cf144b9ce5daea28392ec989a5
Stage20 encoder                      PASS 9883fe2b27020aaff707a753123b35eb3223d21abf295d056ec233e532f94222
Stage20 compact loader               PASS a1ba15881afeb1cf4de9225a06df9ae676b95f596c8ddced7734a445ba7624d0
Stage20 architecture lock            PASS d3bba4d9d9df4432383a7f4239a8562f484cb5805b66d52794873bfaca837b3b
Stage26 measurement protocol         PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
Stage26-4B3 raw timing manifest      PASS 39900c940ca25f3a541efe0dcb5f08d51408581f1f6f4b

In [7]:
# =============================================================================
# STAGE26-4C1A
# PRE-EXECUTION LABEL-FREE EQUIVALENCE ERRATUM
#
# DURABLE PARENT:
#   ce6f51c53acf5b6e64d0f51bc3c28d9b4b12c3a6
#
# DISCOVERED BEFORE ANY EQUIVALENCE / TIMING:
#
#   Historical Stage20CompactCorpus:
#
#     __init__()
#         requires labels.npy
#
#     reconstruct(index)
#         constructs:
#             image        uint8 [64,256]
#             padding_mask bool  [64,256]
#
#         then accesses self.labels[index]
#         solely for the third returned object:
#
#             (image, padding_mask, label)
#
#   Frozen Stage26-4C1 requires:
#       labels_may_be_opened = FALSE
#
# Therefore the v1 equivalence worker cannot be executed without violating
# the no-label scientific boundary.
#
# CORRECTION:
#   - preserve original ce6f51c lock unchanged;
#   - create v2 worker in a separate ERRATUM checkpoint;
#   - bypass historical __init__ during equivalence only;
#   - bind the exact representation arrays from RepresentationSource;
#   - provide an in-memory label-neutral shim returning uint8(0) only to
#     satisfy the historical method's unused third return;
#   - compare ONLY the first two exact historical outputs:
#         image, padding_mask
#
# THE SHIM:
#   - does NOT read labels.npy;
#   - is NOT used for selection;
#   - is NOT used for prediction;
#   - cannot affect image or padding-mask construction;
#   - only permits the historical method to complete its third-output path.
#
# NOTHING ELSE CHANGES:
#   - sample
#   - batches
#   - warmups
#   - timed iterations
#   - CPU affinity
#   - environment gate
#   - timeout
#   - representation boundary
#
# THIS CELL:
#   - performs AST proof of the historical dependency;
#   - creates corrected v2 worker;
#   - freezes an explicit pre-execution erratum;
#   - commits + pushes + remotely byte-verifies it.
#
# NO:
#   - Release extraction
#   - representation reconstruction/materialization
#   - equivalence execution
#   - timing
#   - labels access
#   - PCAP
#   - models
#   - GPU
# =============================================================================

from __future__ import annotations

import ast
import os
import json
import stat
import hashlib
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "ce6f51c53acf5b6e64d0f51bc3c28d9b4b12c3a6"
)

COMMIT_SUBJECT = (
    "stage26: correct label-free representation equivalence"
)


# -----------------------------------------------------------------------------
# Original remotely frozen 4C1 lock
# -----------------------------------------------------------------------------

LOCK_V1 = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c_representation_protocol_lock"
)

V1_WORKER = (
    LOCK_V1
    / "stage26_representation_worker_v1.py"
)

V1_PROTOCOL = (
    LOCK_V1
    / "stage26_representation_protocol.json"
)

V1_BOUNDARY = (
    LOCK_V1
    / "stage26_representation_boundary_map.json"
)

V1_PROVENANCE = (
    LOCK_V1
    / "stage26_4c_upstream_provenance.json"
)

V1_RECEIPT = (
    LOCK_V1
    / "stage26_4c1_representation_protocol_freeze_receipt.json"
)

V1_MANIFEST = (
    LOCK_V1
    / "stage26_4c_representation_lock_manifest.json"
)

V1_INVENTORY = (
    LOCK_V1
    / "stage26_4c0_representation_implementation_inventory.json"
)


EXPECTED_V1_WORKER_SHA256 = (
    "1189f5949e0a0fb0b708449e13cfb172b2ff144d8d724c61c98621d382f57d7a"
)

EXPECTED_V1_PROTOCOL_SHA256 = (
    "4c446212f1af49a6c70c3ef6fcd22facb3f67c650ce6f12a4790807bd58bcb2f"
)

EXPECTED_V1_BOUNDARY_SHA256 = (
    "88efc94664d9658b1e0601506ac4ea369300a98a05049e79280a77107c0828ba"
)

EXPECTED_V1_PROVENANCE_SHA256 = (
    "d63af99619353c4cdf5b9903d97176a4b34c72ce2dde7f51e26090796fc69035"
)

EXPECTED_V1_RECEIPT_SHA256 = (
    "b06e6cebb4123d7ea76225abbf906e2a68cd6364ed584b819973ea87b6e0a2e2"
)

EXPECTED_V1_MANIFEST_SHA256 = (
    "f15f6881caa6fb99de07a979d4c934ce805af4cdf1a7d68b79ebabb9a21a6497"
)

EXPECTED_INVENTORY_SHA256 = (
    "b32a2a21407261d0c3628e80d1c39aab1b6921cf144b9ce5daea28392ec989a5"
)


# -----------------------------------------------------------------------------
# Historical loader
# -----------------------------------------------------------------------------

HISTORICAL_LOADER = (
    REPO
    / "scripts"
    / "stage20_compact_corpus.py"
)

EXPECTED_HISTORICAL_LOADER_SHA256 = (
    "a1ba15881afeb1cf4de9225a06df9ae676b95f596c8ddced7734a445ba7624d0"
)

EXPECTED_RECONSTRUCT_SOURCE_SHA256 = (
    "6290b1427ba42a34590ed7847307e7926310d50056afaa55e83713bdf9afce0c"
)


# =============================================================================
# 1. ERRATUM PACKAGE
# =============================================================================

ERRATUM_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_4c1a_label_free_equivalence_erratum"
)

ERRATUM_DIR = (
    REPO
    / ERRATUM_REL
)

WORKER_V2 = (
    ERRATUM_DIR
    / "stage26_representation_worker_v2.py"
)

PROOF_PATH = (
    ERRATUM_DIR
    / "stage26_4c1a_historical_label_dependency_proof.json"
)

ERRATUM_PATH = (
    ERRATUM_DIR
    / "stage26_4c1a_equivalence_erratum.json"
)

RECEIPT_PATH = (
    ERRATUM_DIR
    / "stage26_4c1a_equivalence_erratum_receipt.json"
)

MANIFEST_PATH = (
    ERRATUM_DIR
    / "stage26_4c1a_equivalence_erratum_manifest.json"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 122
    )

    print(text)

    print(
        "=" * 122
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        check=True,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_text(
    path,
    text,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        f.write(
            text
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def atomic_json(
    path,
    obj,
):

    atomic_text(
        path,
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
        )
        +
        "\n",
    )


def source_segment(
    source,
    node,
):

    result = ast.get_source_segment(
        source,
        node,
    )

    return (
        result
        if result is not None
        else "<source unavailable>"
    )


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        check=True,
        text=False,
    )

    return bytes(
        p.stdout
    )


# =============================================================================
# 3. DURABLE-PARENT GATE
# =============================================================================

banner(
    "STAGE26-4C1A :: DURABLE PARENT GATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected local Stage26 parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before 4C1A correction."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if ERRATUM_DIR.exists():

    raise RuntimeError(
        "4C1A erratum directory already exists. Do not overwrite."
    )


# =============================================================================
# 4. ORIGINAL 4C1 LOCK MUST REMAIN BYTE-EXACT
# =============================================================================

banner(
    "STAGE26-4C1A :: ORIGINAL 4C1 LOCK IDENTITY"
)


lock_checks = [
    (
        "inventory",
        V1_INVENTORY,
        EXPECTED_INVENTORY_SHA256,
    ),
    (
        "worker v1",
        V1_WORKER,
        EXPECTED_V1_WORKER_SHA256,
    ),
    (
        "protocol v1",
        V1_PROTOCOL,
        EXPECTED_V1_PROTOCOL_SHA256,
    ),
    (
        "boundary",
        V1_BOUNDARY,
        EXPECTED_V1_BOUNDARY_SHA256,
    ),
    (
        "provenance",
        V1_PROVENANCE,
        EXPECTED_V1_PROVENANCE_SHA256,
    ),
    (
        "freeze receipt",
        V1_RECEIPT,
        EXPECTED_V1_RECEIPT_SHA256,
    ),
    (
        "lock manifest",
        V1_MANIFEST,
        EXPECTED_V1_MANIFEST_SHA256,
    ),
    (
        "historical loader",
        HISTORICAL_LOADER,
        EXPECTED_HISTORICAL_LOADER_SHA256,
    ),
]


for label, path, expected in lock_checks:

    if not path.exists():

        raise FileNotFoundError(
            path
        )

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )

    print(
        f"{label:26s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )

    if not passed:

        raise RuntimeError(
            f"Frozen identity changed: {label}"
        )


# =============================================================================
# 5. VERIFY ORIGINAL PROTOCOL HAS THE CONFLICT
# =============================================================================

banner(
    "STAGE26-4C1A :: FROZEN NO-LABEL RULE"
)


protocol_v1 = json.loads(
    V1_PROTOCOL.read_text(
        encoding="utf-8"
    )
)

receipt_v1 = json.loads(
    V1_RECEIPT.read_text(
        encoding="utf-8"
    )
)


protocol_checks = {
    "protocol_frozen":
        (
            protocol_v1[
                "status"
            ]
            ==
            "FROZEN_BEFORE_FIRST_REPRESENTATION_MEASUREMENT"
        ),

    "labels_may_not_be_opened":
        (
            protocol_v1[
                "scientific_rules"
            ][
                "labels_may_be_opened"
            ]
            is False
        ),

    "timing_not_yet_performed":
        (
            receipt_v1[
                "scientific_state"
            ][
                "representation_timing_performed"
            ]
            is False
        ),

    "equivalence_not_yet_performed":
        (
            receipt_v1[
                "scientific_state"
            ][
                "equivalence_executed"
            ]
            is False
        ),

    "labels_not_accessed":
        (
            receipt_v1[
                "scientific_state"
            ][
                "labels_accessed"
            ]
            is False
        ),
}


for name, passed in protocol_checks.items():

    print(
        f"{name:40s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    protocol_checks.values()
):

    raise RuntimeError(
        "Original 4C1 pre-execution state is not suitable for erratum."
    )


# =============================================================================
# 6. AST PROOF OF HISTORICAL LABEL DEPENDENCY
# =============================================================================

banner(
    "STAGE26-4C1A :: HISTORICAL LABEL-DEPENDENCY PROOF"
)


loader_text = HISTORICAL_LOADER.read_text(
    encoding="utf-8"
)

loader_tree = ast.parse(
    loader_text
)


class_nodes = [
    node
    for node in loader_tree.body
    if (
        isinstance(
            node,
            ast.ClassDef,
        )
        and
        node.name
        ==
        "Stage20CompactCorpus"
    )
]


if len(
    class_nodes
) != 1:

    raise RuntimeError(
        "Stage20CompactCorpus class not uniquely resolved."
    )


class_node = class_nodes[
    0
]


init_nodes = [
    node
    for node in class_node.body
    if (
        isinstance(
            node,
            ast.FunctionDef,
        )
        and
        node.name
        ==
        "__init__"
    )
]

reconstruct_nodes = [
    node
    for node in class_node.body
    if (
        isinstance(
            node,
            ast.FunctionDef,
        )
        and
        node.name
        ==
        "reconstruct"
    )
]


if len(
    init_nodes
) != 1 or len(
    reconstruct_nodes
) != 1:

    raise RuntimeError(
        "Historical loader methods not uniquely resolved."
    )


init_node = init_nodes[
    0
]

reconstruct_node = reconstruct_nodes[
    0
]


init_source = source_segment(
    loader_text,
    init_node,
)

reconstruct_source = source_segment(
    loader_text,
    reconstruct_node,
)


reconstruct_sha = hashlib.sha256(
    reconstruct_source.encode(
        "utf-8"
    )
).hexdigest()


print(
    "reconstruct SHA256:"
)

print(
    " ",
    reconstruct_sha
)


if reconstruct_sha != EXPECTED_RECONSTRUCT_SOURCE_SHA256:

    raise RuntimeError(
        "Historical reconstruct() source changed."
    )


# -----------------------------------------------------------------------------
# Confirm __init__ constructs labels.npy and loads labels.
# -----------------------------------------------------------------------------

init_has_labels_filename = (
    '"labels.npy"'
    in
    init_source
)

init_has_self_labels = (
    "self.labels"
    in
    init_source
)

init_has_np_load = (
    "np.load"
    in
    init_source
)


print(
    "__init__ contains labels.npy:",
    init_has_labels_filename
)

print(
    "__init__ contains self.labels:",
    init_has_self_labels
)

print(
    "__init__ contains np.load     :",
    init_has_np_load
)


if not all(
    [
        init_has_labels_filename,
        init_has_self_labels,
        init_has_np_load,
    ]
):

    raise RuntimeError(
        "Could not prove historical __init__ label-file dependency."
    )


# -----------------------------------------------------------------------------
# Resolve every self.<attribute> used by reconstruct().
# -----------------------------------------------------------------------------

self_attributes = sorted(
    {
        node.attr
        for node in ast.walk(
            reconstruct_node
        )
        if (
            isinstance(
                node,
                ast.Attribute,
            )
            and
            isinstance(
                node.value,
                ast.Name,
            )
            and
            node.value.id
            ==
            "self"
        )
    }
)


print(
    "\nself attributes used by reconstruct():"
)

for name in self_attributes:

    print(
        " ",
        name
    )


EXPECTED_RECONSTRUCT_SELF_ATTRIBUTES = {
    "_n",
    "packet_lengths",
    "flow_offsets",
    "encoded_bytes",
    "labels",
}


if set(
    self_attributes
) != EXPECTED_RECONSTRUCT_SELF_ATTRIBUTES:

    raise RuntimeError(
        "Unexpected historical reconstruct() object dependency set."
    )


# -----------------------------------------------------------------------------
# Find first self.labels access.
# -----------------------------------------------------------------------------

label_access_lines = sorted(
    {
        int(
            node.lineno
        )
        for node in ast.walk(
            reconstruct_node
        )
        if (
            isinstance(
                node,
                ast.Attribute,
            )
            and
            isinstance(
                node.value,
                ast.Name,
            )
            and
            node.value.id
            ==
            "self"
            and
            node.attr
            ==
            "labels"
        )
    }
)


if not label_access_lines:

    raise RuntimeError(
        "Historical reconstruct() does not access self.labels as expected."
    )


first_label_access_line = min(
    label_access_lines
)


# -----------------------------------------------------------------------------
# Find image / padding-mask subscript-write lines.
# -----------------------------------------------------------------------------

representation_write_lines = []


for node in ast.walk(
    reconstruct_node
):

    if not isinstance(
        node,
        ast.Assign,
    ):

        continue

    for target in node.targets:

        if not isinstance(
            target,
            ast.Subscript,
        ):

            continue

        if not isinstance(
            target.value,
            ast.Name,
        ):

            continue

        if target.value.id in {
            "image",
            "padding_mask",
        }:

            representation_write_lines.append(
                int(
                    node.lineno
                )
            )


if not representation_write_lines:

    raise RuntimeError(
        "Could not resolve historical image/mask write locations."
    )


last_representation_write_line = max(
    representation_write_lines
)


print(
    "\nLast image/mask write line:",
    last_representation_write_line
)

print(
    "First self.labels access line:",
    first_label_access_line
)


if not (
    first_label_access_line
    >
    last_representation_write_line
):

    raise RuntimeError(
        "Label access occurs before representation construction is complete."
    )


# -----------------------------------------------------------------------------
# Verify return order = image, padding_mask, label.
# -----------------------------------------------------------------------------

return_tuples = []


for node in ast.walk(
    reconstruct_node
):

    if not isinstance(
        node,
        ast.Return,
    ):

        continue

    if not isinstance(
        node.value,
        ast.Tuple,
    ):

        continue

    names = []

    valid = True

    for element in node.value.elts:

        if not isinstance(
            element,
            ast.Name,
        ):

            valid = False
            break

        names.append(
            element.id
        )

    if valid:

        return_tuples.append(
            names
        )


print(
    "\nReturn tuples:",
    return_tuples
)


if return_tuples != [
    [
        "image",
        "padding_mask",
        "label",
    ]
]:

    raise RuntimeError(
        "Unexpected historical reconstruct() return contract."
    )


proof = {
    "schema":
        "stage26_4c1a_historical_label_dependency_proof_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "historical_loader_path":
        str(
            HISTORICAL_LOADER.relative_to(
                REPO
            )
        ),

    "historical_loader_sha256":
        EXPECTED_HISTORICAL_LOADER_SHA256,

    "reconstruct_source_sha256":
        reconstruct_sha,

    "historical_class":
        "Stage20CompactCorpus",

    "init_requires_labels_file":
        True,

    "init_constructs_labels_filename":
        init_has_labels_filename,

    "init_uses_np_load":
        init_has_np_load,

    "reconstruct_self_attributes":
        self_attributes,

    "representation_write_lines":
        sorted(
            representation_write_lines
        ),

    "last_representation_write_line":
        last_representation_write_line,

    "first_label_access_line":
        first_label_access_line,

    "label_access_occurs_after_representation_writes":
        (
            first_label_access_line
            >
            last_representation_write_line
        ),

    "return_order": [
        "image",
        "padding_mask",
        "label",
    ],

    "first_two_outputs_are_representation_outputs":
        True,

    "third_output_is_label":
        True,

    "scientific_conclusion":
        (
            "Historical reconstruct() can be used to validate its first two "
            "representation outputs without opening labels.npy by bypassing "
            "__init__, binding the exact representation arrays, and supplying "
            "a valid in-memory label-neutral shim solely for the third output."
        ),
}


# =============================================================================
# 7. BUILD ERRATUM DIRECTORY
# =============================================================================

ERRATUM_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


atomic_json(
    PROOF_PATH,
    proof,
)


proof_sha = sha256_file(
    PROOF_PATH
)


print(
    "\nDependency proof SHA256:"
)

print(
    " ",
    proof_sha
)


# =============================================================================
# 8. CREATE CORRECTED WORKER V2 BY NARROW PATCH
# =============================================================================

banner(
    "STAGE26-4C1A :: CREATE CORRECTED WORKER V2"
)


worker_v1_text = V1_WORKER.read_text(
    encoding="utf-8"
)


# -----------------------------------------------------------------------------
# Insert label-neutral shim before run_equivalence().
# -----------------------------------------------------------------------------

shim_marker = (
    "\ndef run_equivalence(\n"
)


if worker_v1_text.count(
    shim_marker
) != 1:

    raise RuntimeError(
        "Could not uniquely locate run_equivalence() insertion point."
    )


shim_text = r'''

class _LabelNeutralValidationShim:
    """
    No-I/O compatibility object for the historical reconstruct() third output.

    Stage26 equivalence compares ONLY historical outputs 0 and 1:
        image, padding_mask

    The historical method nevertheless reads self.labels[index] after those
    representation outputs have already been constructed. Returning uint8(0)
    allows that unrelated third-output path to complete without opening
    labels.npy.
    """

    def __getitem__(
        self,
        index,
    ):

        _ = int(
            index
        )

        return np.uint8(
            0
        )


'''


worker_v2_text = worker_v1_text.replace(
    shim_marker,
    shim_text
    +
    "def run_equivalence(\n",
)


# -----------------------------------------------------------------------------
# Replace historical __init__ invocation only.
# -----------------------------------------------------------------------------

old_instantiation = '''    historical = historical_class(
        Path(
            cfg[
                "corpus_dir"
            ]
        )
    )
'''


new_instantiation = '''    # ---------------------------------------------------------------------
    # LABEL-FREE HISTORICAL EQUIVALENCE ADAPTER
    #
    # Do NOT invoke Stage20CompactCorpus.__init__ because the historical
    # constructor requires labels.npy. The frozen Stage26 representation
    # boundary explicitly forbids label access.
    #
    # reconstruct() uses exactly five object attributes:
    #   _n, packet_lengths, flow_offsets, encoded_bytes, labels
    #
    # The first four are bound directly to this worker's exact authoritative
    # representation source. The fifth is a no-I/O neutral shim used solely
    # by the historical method's third output after image/mask construction.
    # ---------------------------------------------------------------------

    historical = historical_class.__new__(
        historical_class
    )

    historical._n = source.n

    historical.packet_lengths = (
        source.packet_lengths
    )

    historical.flow_offsets = (
        source.flow_offsets
    )

    historical.encoded_bytes = (
        source.encoded_bytes
    )

    historical.labels = (
        _LabelNeutralValidationShim()
    )
'''


count_old = worker_v2_text.count(
    old_instantiation
)


print(
    "Historical constructor block matches:",
    count_old
)


if count_old != 1:

    raise RuntimeError(
        "Could not uniquely patch historical constructor invocation."
    )


worker_v2_text = worker_v2_text.replace(
    old_instantiation,
    new_instantiation,
)


# -----------------------------------------------------------------------------
# Add explicit equivalence provenance fields.
# -----------------------------------------------------------------------------

old_equiv_tail = '''        "timing_performed":
            False,

        "gpu_used":
            False,
    }
'''


new_equiv_tail = '''        "timing_performed":
            False,

        "historical_constructor_invoked":
            False,

        "historical_labels_file_opened":
            False,

        "label_neutral_validation_shim_used":
            True,

        "historical_outputs_compared": [
            "image",
            "padding_mask",
        ],

        "historical_label_output_compared":
            False,

        "gpu_used":
            False,
    }
'''


if worker_v2_text.count(
    old_equiv_tail
) != 1:

    raise RuntimeError(
        "Could not uniquely patch equivalence-result provenance."
    )


worker_v2_text = worker_v2_text.replace(
    old_equiv_tail,
    new_equiv_tail,
)


# Syntax validation without pycache.
compile(
    worker_v2_text,
    str(
        WORKER_V2
    ),
    "exec",
)


atomic_text(
    WORKER_V2,
    worker_v2_text,
)


worker_v2_sha = sha256_file(
    WORKER_V2
)


print(
    "Worker v2 syntax : PASS"
)

print(
    "Worker v1 SHA256 :",
    EXPECTED_V1_WORKER_SHA256
)

print(
    "Worker v2 SHA256 :",
    worker_v2_sha
)


if worker_v2_sha == EXPECTED_V1_WORKER_SHA256:

    raise RuntimeError(
        "Worker v2 unexpectedly identical to v1."
    )


# =============================================================================
# 9. STATIC V2 SAFETY AUDIT
# =============================================================================

banner(
    "STAGE26-4C1A :: V2 STATIC SAFETY AUDIT"
)


worker_v2_tree = ast.parse(
    worker_v2_text
)


imports = set()


for node in ast.walk(
    worker_v2_tree
):

    if isinstance(
        node,
        ast.Import,
    ):

        for alias in node.names:

            imports.add(
                alias.name.split(
                    "."
                )[
                    0
                ]
            )


    elif isinstance(
        node,
        ast.ImportFrom,
    ):

        if node.module:

            imports.add(
                node.module.split(
                    "."
                )[
                    0
                ]
            )


for forbidden in {
    "scapy",
    "dpkt",
    "pyshark",
    "torch",
}:

    if forbidden in imports:

        raise RuntimeError(
            f"Forbidden import in worker v2: {forbidden}"
        )


# v2 itself must not construct/open a labels.npy path.
if (
    ' / "labels.npy"' in worker_v2_text
    or
    "/ 'labels.npy'" in worker_v2_text
):

    raise RuntimeError(
        "Worker v2 constructs labels.npy path."
    )


required_v2_tokens = [
    "historical_class.__new__",
    "_LabelNeutralValidationShim",
    "historical_constructor_invoked",
    "historical_labels_file_opened",
    "historical_outputs_compared",
]


for token in required_v2_tokens:

    passed = (
        token
        in
        worker_v2_text
    )

    print(
        f"{token:40s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

    if not passed:

        raise RuntimeError(
            f"Required v2 mechanism absent: {token}"
        )


print(
    "\nlabels.npy path construction in v2: ABSENT"
)

print(
    "Representation timing performed       : NO"
)

print(
    "Equivalence performed                  : NO"
)


# =============================================================================
# 10. FREEZE PRE-EXECUTION ERRATUM
# =============================================================================

banner(
    "STAGE26-4C1A :: FREEZE ERRATUM"
)


erratum = {
    "schema":
        "stage26_4c1a_label_free_equivalence_erratum_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4C1A",

    "status":
        "PRE_EXECUTION_CORRECTION_FROZEN",

    "scientific_parent":
        EXPECTED_PARENT,

    "original_lock": {
        "commit":
            EXPECTED_PARENT,

        "worker_v1_sha256":
            EXPECTED_V1_WORKER_SHA256,

        "protocol_v1_sha256":
            EXPECTED_V1_PROTOCOL_SHA256,

        "boundary_sha256":
            EXPECTED_V1_BOUNDARY_SHA256,

        "provenance_sha256":
            EXPECTED_V1_PROVENANCE_SHA256,

        "freeze_receipt_sha256":
            EXPECTED_V1_RECEIPT_SHA256,

        "manifest_sha256":
            EXPECTED_V1_MANIFEST_SHA256,
    },

    "discovery": {
        "historical_loader_sha256":
            EXPECTED_HISTORICAL_LOADER_SHA256,

        "historical_reconstruct_source_sha256":
            EXPECTED_RECONSTRUCT_SOURCE_SHA256,

        "historical_constructor_requires_labels_npy":
            True,

        "historical_reconstruct_reads_label":
            True,

        "historical_return_order": [
            "image",
            "padding_mask",
            "label",
        ],

        "label_access_after_representation_construction":
            True,

        "proof_path":
            str(
                PROOF_PATH.relative_to(
                    REPO
                )
            ),

        "proof_sha256":
            proof_sha,
    },

    "conflict": {
        "original_protocol_labels_may_be_opened":
            False,

        "worker_v1_would_invoke_historical_constructor":
            True,

        "worker_v1_executable_without_labels_file":
            False,

        "reason_for_correction":
            (
                "Executing worker v1 equivalence would require labels.npy "
                "through the historical constructor, violating the already-"
                "frozen no-label representation boundary."
            ),
    },

    "correction": {
        "type":
            "PRE_EXECUTION_IMPLEMENTATION_ERRATUM",

        "worker_v2_path":
            str(
                WORKER_V2.relative_to(
                    REPO
                )
            ),

        "worker_v2_sha256":
            worker_v2_sha,

        "historical_constructor_bypassed":
            True,

        "historical_reconstruct_method_itself_used":
            True,

        "representation_arrays_bound_from_exact_authoritative_source":
            True,

        "in_memory_label_neutral_shim_used":
            True,

        "label_neutral_shim_value":
            0,

        "label_neutral_shim_dtype":
            "uint8",

        "labels_file_opened":
            False,

        "historical_outputs_compared": [
            "image",
            "padding_mask",
        ],

        "historical_label_output_compared":
            False,

        "scientific_reason":
            (
                "The label is the historical method's third output and is "
                "read only after image and padding-mask construction. "
                "A valid zero shim allows the method to return while leaving "
                "the first two representation outputs dependent only on the "
                "exact compact-corpus representation arrays."
            ),
    },

    "unchanged_frozen_science": {
        "representation_boundary_changed":
            False,

        "image_shape":
            [
                64,
                256,
            ],

        "padding_mask_shape":
            [
                64,
                256,
            ],

        "sample_start_index_zero_based":
            26042,

        "largest_sample_end_index_exclusive":
            34234,

        "batch_sizes":
            [
                1,
                64,
                256,
                1024,
                8192,
            ],

        "CPU_affinity":
            [
                0
            ],

        "thread_count":
            1,

        "condition_timeout_seconds":
            600,

        "equivalence_start_index_zero_based":
            26042,

        "equivalence_flow_count":
            128,

        "equivalence_is_timed":
            False,

        "timing_policy_changed":
            False,

        "sample_selection_changed":
            False,

        "labels_may_be_opened":
            False,

        "release_corpus_regeneration_forbidden":
            True,

        "complete_E2E_claim_allowed":
            False,

        "GPU":
            False,
    },

    "effective_execution_rule": {
        "equivalence_worker":
            "stage26_representation_worker_v2.py",

        "benchmark_worker":
            "stage26_representation_worker_v2.py",

        "original_worker_v1_superseded_for_future_execution":
            True,

        "original_4C1_lock_remains_immutable_in_git_history":
            True,

        "timing_still_forbidden_until_equivalence_pass_and_git_anchor":
            True,
    },

    "scientific_state_at_erratum_freeze": {
        "equivalence_executed":
            False,

        "representation_timing_performed":
            False,

        "representation_materialized":
            False,

        "labels_accessed":
            False,

        "pcap_accessed":
            False,

        "corpus_regenerated":
            False,

        "models_loaded":
            False,

        "gpu_used":
            False,
    },
}


atomic_json(
    ERRATUM_PATH,
    erratum,
)


erratum_sha = sha256_file(
    ERRATUM_PATH
)


print(
    "Erratum SHA256:"
)

print(
    " ",
    erratum_sha
)


# =============================================================================
# 11. ERRATUM RECEIPT
# =============================================================================

receipt = {
    "schema":
        "stage26_4c1a_equivalence_erratum_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4C1A",

    "status":
        "LABEL_FREE_EQUIVALENCE_CORRECTION_FROZEN_NOT_EXECUTED",

    "parent_commit":
        EXPECTED_PARENT,

    "historical_dependency_proof_sha256":
        proof_sha,

    "worker_v2_sha256":
        worker_v2_sha,

    "erratum_sha256":
        erratum_sha,

    "correction_performed_before_equivalence":
        True,

    "correction_performed_before_representation_timing":
        True,

    "sample_changed":
        False,

    "timing_policy_changed":
        False,

    "representation_boundary_changed":
        False,

    "labels_accessed":
        False,

    "equivalence_executed":
        False,

    "representation_timing_performed":
        False,

    "pcap_accessed":
        False,

    "release_corpus_regenerated":
        False,

    "models_loaded":
        False,

    "gpu_used":
        False,

    "next_action":
        (
            "Restore only encoded_bytes.bin, flow_offsets.npy, and "
            "packet_lengths.npy from the authoritative Release; verify their "
            "hashes; then execute worker v2 VALIDATE_EQUIVALENCE on the frozen "
            "128-flow range. Do not perform representation timing until that "
            "equivalence result is separately Git-anchored."
        ),
}


atomic_json(
    RECEIPT_PATH,
    receipt,
)


receipt_sha = sha256_file(
    RECEIPT_PATH
)


# =============================================================================
# 12. BUILD ERRATUM MANIFEST
# =============================================================================

banner(
    "STAGE26-4C1A :: BUILD MANIFEST"
)


package_files = [
    WORKER_V2,
    PROOF_PATH,
    ERRATUM_PATH,
    RECEIPT_PATH,
]


manifest_rows = []


for path in package_files:

    manifest_rows.append(
        {
            "repo_relative_path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest = {
    "schema":
        "stage26_4c1a_equivalence_erratum_manifest_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        "READY_FOR_GIT_ANCHOR",

    "parent_commit":
        EXPECTED_PARENT,

    "original_4c1_protocol_sha256":
        EXPECTED_V1_PROTOCOL_SHA256,

    "original_worker_v1_sha256":
        EXPECTED_V1_WORKER_SHA256,

    "worker_v2_sha256":
        worker_v2_sha,

    "dependency_proof_sha256":
        proof_sha,

    "erratum_sha256":
        erratum_sha,

    "receipt_sha256":
        receipt_sha,

    "file_count_excluding_manifest":
        len(
            manifest_rows
        ),

    "files":
        manifest_rows,
}


atomic_json(
    MANIFEST_PATH,
    manifest,
)


manifest_sha = sha256_file(
    MANIFEST_PATH
)


print(
    "Worker v2 :",
    worker_v2_sha
)

print(
    "Proof     :",
    proof_sha
)

print(
    "Erratum   :",
    erratum_sha
)

print(
    "Receipt   :",
    receipt_sha
)

print(
    "Manifest  :",
    manifest_sha
)


# =============================================================================
# 13. LOCAL PACKAGE AUDIT
# =============================================================================

banner(
    "STAGE26-4C1A :: LOCAL PACKAGE AUDIT"
)


for row in manifest_rows:

    path = (
        REPO
        / row[
            "repo_relative_path"
        ]
    )

    actual_size = int(
        path.stat().st_size
    )

    actual_sha = sha256_file(
        path
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Local erratum package audit failed."
        )


# =============================================================================
# 14. GIT CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-4C1A :: GIT CHANGE AUDIT"
)


repo_status = git(
    "status",
    "--porcelain",
)


print(
    repo_status
)


if not repo_status:

    raise RuntimeError(
        "Expected uncommitted 4C1A erratum."
    )


unexpected = []


for line in repo_status.splitlines():

    rel = line[
        3:
    ]


    if not rel.startswith(
        str(
            ERRATUM_REL
        )
        +
        "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository modifications:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 15. RESTORE GIT IDENTITY
# =============================================================================

author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_PARENT,
)

author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_PARENT,
)


if (
    not author_name.strip()
    or
    "@"
    not in
    author_email
):

    raise RuntimeError(
        "Could not resolve Git identity."
    )


git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


print(
    "\nGit author:"
)

print(
    " ",
    author_name,
    "<" + author_email + ">"
)


# =============================================================================
# 16. COMMIT
# =============================================================================

banner(
    "STAGE26-4C1A :: COMMIT"
)


git(
    "add",
    str(
        ERRATUM_REL
    ),
)


staged = git(
    "diff",
    "--cached",
    "--name-status",
)


print(
    staged
)


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "4C1A commit parent mismatch."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "4C1A commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository not clean after 4C1A commit."
    )


# =============================================================================
# 17. PUSH
# =============================================================================

banner(
    "STAGE26-4C1A :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    pushed = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
        check=True,
        text=True,
    )


    print(
        pushed.stdout.strip()
    )


github_token = None


# =============================================================================
# 18. REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-4C1A :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "origin/main != 4C1A commit."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote 4C1A parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote 4C1A subject mismatch."
    )


# =============================================================================
# 19. REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-4C1A :: REMOTE BYTE VERIFICATION"
)


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    str(
        MANIFEST_PATH.relative_to(
            REPO
        )
    ),
)


remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "Remote manifest SHA256:"
)

print(
    " ",
    remote_manifest_sha
)


if remote_manifest_sha != manifest_sha:

    raise RuntimeError(
        "Remote 4C1A manifest mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


if (
    remote_manifest[
        "file_count_excluding_manifest"
    ]
    !=
    4
):

    raise RuntimeError(
        "Unexpected 4C1A package file count."
    )


for row in remote_manifest[
    "files"
]:

    data = git_blob_bytes(
        "origin/main",
        row[
            "repo_relative_path"
        ],
    )

    actual_size = len(
        data
    )

    actual_sha = sha256_bytes(
        data
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Remote 4C1A byte verification failed."
        )


# =============================================================================
# 20. REMOTE SCIENTIFIC AUDIT
# =============================================================================

banner(
    "STAGE26-4C1A :: REMOTE SCIENTIFIC AUDIT"
)


remote_erratum = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            ERRATUM_PATH.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_receipt = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            RECEIPT_PATH.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)


remote_checks = {
    "pre_execution":
        (
            remote_erratum[
                "status"
            ]
            ==
            "PRE_EXECUTION_CORRECTION_FROZEN"
        ),

    "original_protocol_preserved":
        (
            remote_erratum[
                "original_lock"
            ][
                "protocol_v1_sha256"
            ]
            ==
            EXPECTED_V1_PROTOCOL_SHA256
        ),

    "worker_v2_exact":
        (
            remote_erratum[
                "correction"
            ][
                "worker_v2_sha256"
            ]
            ==
            worker_v2_sha
        ),

    "historical_constructor_bypassed":
        (
            remote_erratum[
                "correction"
            ][
                "historical_constructor_bypassed"
            ]
            is True
        ),

    "historical_method_used":
        (
            remote_erratum[
                "correction"
            ][
                "historical_reconstruct_method_itself_used"
            ]
            is True
        ),

    "label_file_not_opened":
        (
            remote_erratum[
                "correction"
            ][
                "labels_file_opened"
            ]
            is False
        ),

    "only_representation_outputs_compared":
        (
            remote_erratum[
                "correction"
            ][
                "historical_outputs_compared"
            ]
            ==
            [
                "image",
                "padding_mask",
            ]
        ),

    "label_output_not_compared":
        (
            remote_erratum[
                "correction"
            ][
                "historical_label_output_compared"
            ]
            is False
        ),

    "sample_unchanged":
        (
            remote_erratum[
                "unchanged_frozen_science"
            ][
                "sample_selection_changed"
            ]
            is False
        ),

    "timing_policy_unchanged":
        (
            remote_erratum[
                "unchanged_frozen_science"
            ][
                "timing_policy_changed"
            ]
            is False
        ),

    "boundary_unchanged":
        (
            remote_erratum[
                "unchanged_frozen_science"
            ][
                "representation_boundary_changed"
            ]
            is False
        ),

    "equivalence_not_run":
        (
            remote_receipt[
                "equivalence_executed"
            ]
            is False
        ),

    "timing_not_run":
        (
            remote_receipt[
                "representation_timing_performed"
            ]
            is False
        ),

    "labels_not_accessed":
        (
            remote_receipt[
                "labels_accessed"
            ]
            is False
        ),

    "gpu_false":
        (
            remote_receipt[
                "gpu_used"
            ]
            is False
        ),
}


for name, passed in remote_checks.items():

    print(
        f"{name:44s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    remote_checks.values()
):

    raise RuntimeError(
        "Remote 4C1A scientific audit failed."
    )


# =============================================================================
# 21. FINAL AUDIT
# =============================================================================

banner(
    "STAGE26-4C1A :: FINAL AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != final_remote:

    raise RuntimeError(
        "Local/remote divergence after 4C1A."
    )


if final_status:

    raise RuntimeError(
        "Repository not clean after 4C1A."
    )


# =============================================================================
# 22. CLOSURE
# =============================================================================

banner(
    "STAGE26-4C1A LABEL-FREE EQUIVALENCE ERRATUM COMPLETE"
)


print(
    "NEW DURABLE COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nPARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nDISCOVERED HISTORICAL DEPENDENCY:"
)

print(
    "  Stage20CompactCorpus.__init__ opens labels.npy : YES"
)

print(
    "  reconstruct() reads self.labels                : YES"
)

print(
    "  image/mask construction precedes label access  : YES"
)

print(
    "  historical return order:"
)

print(
    "    image, padding_mask, label"
)


print(
    "\nCORRECTION:"
)

print(
    "  historical __init__ invoked : NO"
)

print(
    "  exact reconstruct() used    : YES"
)

print(
    "  representation arrays       : exact authoritative source"
)

print(
    "  label-neutral shim          : uint8(0), memory-only"
)

print(
    "  labels.npy opened           : NO"
)

print(
    "  outputs compared            : image + padding_mask only"
)


print(
    "\nUNCHANGED SCIENCE:"
)

print(
    "  sample start     : 26042"
)

print(
    "  largest end      : 34234"
)

print(
    "  batches          : [1,64,256,1024,8192]"
)

print(
    "  equivalence flows: 128"
)

print(
    "  equivalence timed: NO"
)

print(
    "  CPU affinity     : [0]"
)

print(
    "  timing policy    : UNCHANGED"
)

print(
    "  boundary         : UNCHANGED"
)


print(
    "\nERRATUM HASHES:"
)

print(
    "  worker v2 :",
    worker_v2_sha
)

print(
    "  proof     :",
    proof_sha
)

print(
    "  erratum   :",
    erratum_sha
)

print(
    "  receipt   :",
    receipt_sha
)

print(
    "  manifest  :",
    manifest_sha
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  PRE-EXECUTION CORRECTION REMOTELY LOCKED : YES"
)

print(
    "  EQUIVALENCE EXECUTED                     : NO"
)

print(
    "  REPRESENTATION TIMING                     : NO"
)

print(
    "  LABELS ACCESSED                           : NO"
)

print(
    "  RELEASE CORPUS REGENERATED                : NO"
)

print(
    "  PCAP ACCESSED                             : NO"
)

print(
    "  MODELS LOADED                             : NO"
)

print(
    "  GPU                                       : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Restore ONLY:")
print(
    "    encoded_bytes.bin")
print(
    "    flow_offsets.npy")
print(
    "    packet_lengths.npy")

print(
    "  from the already-verified Monday Release TAR,"
)

print(
    "  verify their exact SHA256 values, and run the frozen"
)

print(
    "  128-flow UNTIMED equivalence with worker v2."
)

print(
    "  Representation timing remains forbidden until that"
)

print(
    "  equivalence result is separately committed and pushed."
)


STAGE26-4C1A :: DURABLE PARENT GATE
Expected parent: ce6f51c53acf5b6e64d0f51bc3c28d9b4b12c3a6
Local HEAD     : ce6f51c53acf5b6e64d0f51bc3c28d9b4b12c3a6
origin/main    : ce6f51c53acf5b6e64d0f51bc3c28d9b4b12c3a6
Repo clean     : True

STAGE26-4C1A :: ORIGINAL 4C1 LOCK IDENTITY
inventory                  PASS b32a2a21407261d0c3628e80d1c39aab1b6921cf144b9ce5daea28392ec989a5
worker v1                  PASS 1189f5949e0a0fb0b708449e13cfb172b2ff144d8d724c61c98621d382f57d7a
protocol v1                PASS 4c446212f1af49a6c70c3ef6fcd22facb3f67c650ce6f12a4790807bd58bcb2f
boundary                   PASS 88efc94664d9658b1e0601506ac4ea369300a98a05049e79280a77107c0828ba
provenance                 PASS d63af99619353c4cdf5b9903d97176a4b34c72ce2dde7f51e26090796fc69035
freeze receipt             PASS b06e6cebb4123d7ea76225abbf906e2a68cd6364ed584b819973ea87b6e0a2e2
lock manifest              PASS f15f6881caa6fb99de07a979d4c934ce805af4cdf1a7d68b79ebabb9a21a6497
historical loader          PASS a1ba15881afe

In [8]:
# =============================================================================
# STAGE26-4C2
# RESTORE REQUIRED RELEASE MEMBERS
# RUN FROZEN 128-FLOW UNTIMED REPRESENTATION EQUIVALENCE
# COMMIT + PUSH RESULT BEFORE ANY REPRESENTATION TIMING
#
# DURABLE PARENT:
#   6a55aca543111e36e9fe83f16f008932d2392a13
#
# EFFECTIVE WORKER:
#   stage26_representation_worker_v2.py
#
# WORKER SHA256:
#   ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23
#
# FROZEN EQUIVALENCE:
#   start index : 26042
#   flow count  : 128
#   end         : 26170 exclusive
#   CPU affinity: [0]
#   timed       : NO
#
# AUTHORITATIVE INPUT:
#   existing Monday GitHub Release TAR
#
# RESTORE ONLY:
#   encoded_bytes.bin
#   flow_offsets.npy
#   packet_lengths.npy
#
# DO NOT RESTORE:
#   labels.npy
#
# SCIENTIFIC PURPOSE:
#   Demonstrate exact equality between:
#
#       Stage26 representation worker v2
#
#   and
#
#       frozen Stage20CompactCorpus.reconstruct()
#
#   for the representation outputs only:
#
#       image        uint8 [64,256]
#       padding_mask bool  [64,256]
#
# LABEL HANDLING:
#   - labels.npy is not extracted
#   - labels.npy is not opened
#   - historical __init__ is not invoked
#   - memory-only uint8(0) shim satisfies unused historical third output
#   - label output is not compared
#
# THIS CELL DOES:
#   1. verify durable scientific anchor;
#   2. verify frozen protocol/erratum identities;
#   3. restore ONLY the 3 required Release members;
#   4. SHA256-verify each restored member;
#   5. prove labels.npy is absent from runtime corpus directory;
#   6. run worker v2 VALIDATE_EQUIVALENCE in fresh subprocess;
#   7. verify exact result semantics;
#   8. create equivalence receipt + manifest;
#   9. commit + push equivalence checkpoint;
#  10. remotely byte/scientifically verify it.
#
# ABSOLUTE RULES:
#   - NO representation timing.
#   - NO benchmark mode.
#   - NO labels.npy extraction/access.
#   - NO PCAP.
#   - NO corpus regeneration.
#   - NO model loading.
#   - NO inference.
#   - NO Thursday / Friday.
#   - NO GPU.
#
# AFTER SUCCESS:
#   Representation timing becomes scientifically PERMITTED,
#   but is NOT performed by this cell.
# =============================================================================

from __future__ import annotations

import os
import sys
import json
import stat
import shutil
import hashlib
import tarfile
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. FROZEN IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

EXPECTED_PARENT = (
    "6a55aca543111e36e9fe83f16f008932d2392a13"
)

COMMIT_SUBJECT = (
    "stage26: anchor representation equivalence gate"
)


# -----------------------------------------------------------------------------
# Original representation protocol
# -----------------------------------------------------------------------------

LOCK_V1 = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c_representation_protocol_lock"
)

PROTOCOL_V1 = (
    LOCK_V1
    / "stage26_representation_protocol.json"
)

BOUNDARY_V1 = (
    LOCK_V1
    / "stage26_representation_boundary_map.json"
)

EXPECTED_PROTOCOL_V1_SHA256 = (
    "4c446212f1af49a6c70c3ef6fcd22facb3f67c650ce6f12a4790807bd58bcb2f"
)

EXPECTED_BOUNDARY_V1_SHA256 = (
    "88efc94664d9658b1e0601506ac4ea369300a98a05049e79280a77107c0828ba"
)


# -----------------------------------------------------------------------------
# 4C1A effective erratum
# -----------------------------------------------------------------------------

ERRATUM_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c1a_label_free_equivalence_erratum"
)

WORKER_V2 = (
    ERRATUM_DIR
    / "stage26_representation_worker_v2.py"
)

ERRATUM = (
    ERRATUM_DIR
    / "stage26_4c1a_equivalence_erratum.json"
)

ERRATUM_RECEIPT = (
    ERRATUM_DIR
    / "stage26_4c1a_equivalence_erratum_receipt.json"
)

ERRATUM_MANIFEST = (
    ERRATUM_DIR
    / "stage26_4c1a_equivalence_erratum_manifest.json"
)

DEPENDENCY_PROOF = (
    ERRATUM_DIR
    / "stage26_4c1a_historical_label_dependency_proof.json"
)


EXPECTED_WORKER_V2_SHA256 = (
    "ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23"
)

EXPECTED_PROOF_SHA256 = (
    "05ecedbb0d973173c276e5bcec54041538b345614c296561867622933781b597"
)

EXPECTED_ERRATUM_SHA256 = (
    "1f6748b3042b64626de73c74f9a9ff2f8c1f9ce5e6b445a5dd2030137f951839"
)

EXPECTED_ERRATUM_RECEIPT_SHA256 = (
    "3341f747f95b9da36c17c39727aaa139a71858551c2cadc31c233454a98f8ddb"
)

EXPECTED_ERRATUM_MANIFEST_SHA256 = (
    "b3dd2844829930f016f760805d75f066795ecb9f86b4b52a668dee4beea2470e"
)


# -----------------------------------------------------------------------------
# Historical loader
# -----------------------------------------------------------------------------

HISTORICAL_LOADER = (
    REPO
    / "scripts"
    / "stage20_compact_corpus.py"
)

EXPECTED_HISTORICAL_LOADER_SHA256 = (
    "a1ba15881afeb1cf4de9225a06df9ae676b95f596c8ddced7734a445ba7624d0"
)

HISTORICAL_LOADER_CLASS = (
    "Stage20CompactCorpus"
)

EXPECTED_RECONSTRUCT_SOURCE_SHA256 = (
    "6290b1427ba42a34590ed7847307e7926310d50056afaa55e83713bdf9afce0c"
)


# -----------------------------------------------------------------------------
# Authoritative existing Release TAR
# -----------------------------------------------------------------------------

MONDAY_TAR = (
    STAGE26_ROOT
    / "release_corpora"
    / "Monday"
    / "stage20-Monday-compact-corpus-v1.tar"
)

EXPECTED_TAR_BYTES = (
    595_261_440
)

EXPECTED_TAR_SHA256 = (
    "4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20"
)


REQUIRED_MEMBERS = {
    "encoded_bytes.bin": {
        "tar_member":
            "Monday/encoded_bytes.bin",

        "size_bytes":
            522_845_159,

        "sha256":
            "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",
    },

    "flow_offsets.npy": {
        "tar_member":
            "Monday/flow_offsets.npy",

        "size_bytes":
            4_228_208,

        "sha256":
            "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",
    },

    "packet_lengths.npy": {
        "tar_member":
            "Monday/packet_lengths.npy",

        "size_bytes":
            67_649_280,

        "sha256":
            "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
    },
}


FORBIDDEN_MEMBER = (
    "labels.npy"
)

FORBIDDEN_TAR_MEMBER = (
    "Monday/labels.npy"
)


# =============================================================================
# 1. FROZEN EQUIVALENCE CONDITION
# =============================================================================

FLOW_COUNT = (
    528_509
)

START_INDEX = (
    26_042
)

EQUIVALENCE_FLOW_COUNT = (
    128
)

END_INDEX_EXCLUSIVE = (
    START_INDEX
    +
    EQUIVALENCE_FLOW_COUNT
)

CPU_AFFINITY = [
    0
]


# =============================================================================
# 2. RUNTIME PATHS
# =============================================================================

RUNTIME_ROOT = (
    STAGE26_ROOT
    / "representation"
    / "stage26_4c2_equivalence"
)

CORPUS_RUNTIME_DIR = (
    RUNTIME_ROOT
    / "Monday_release_representation_subset"
)

CONFIG_PATH = (
    RUNTIME_ROOT
    / "stage26_4c2_equivalence_config.json"
)

RESULT_PATH = (
    RUNTIME_ROOT
    / "stage26_4c2_equivalence_result.json"
)

RESTORATION_RECEIPT_PATH = (
    RUNTIME_ROOT
    / "stage26_4c2_release_member_restoration_receipt.json"
)

EQUIVALENCE_RECEIPT_RUNTIME = (
    RUNTIME_ROOT
    / "stage26_4c2_equivalence_receipt.json"
)


# =============================================================================
# 3. DURABLE CHECKPOINT PATHS
# =============================================================================

CHECKPOINT_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_4c2_representation_equivalence"
)

CHECKPOINT_DIR = (
    REPO
    / CHECKPOINT_REL
)

CHECKPOINT_CONFIG = (
    CHECKPOINT_DIR
    / CONFIG_PATH.name
)

CHECKPOINT_RESULT = (
    CHECKPOINT_DIR
    / RESULT_PATH.name
)

CHECKPOINT_RESTORATION = (
    CHECKPOINT_DIR
    / RESTORATION_RECEIPT_PATH.name
)

CHECKPOINT_RECEIPT = (
    CHECKPOINT_DIR
    / EQUIVALENCE_RECEIPT_RUNTIME.name
)

CHECKPOINT_MANIFEST = (
    CHECKPOINT_DIR
    / "stage26_4c2_representation_equivalence_manifest.json"
)


# =============================================================================
# 4. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 122
    )

    print(text)

    print(
        "=" * 122
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
    timeout=None,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
        timeout=timeout,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        check=True,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        check=True,
        text=False,
    )

    return bytes(
        p.stdout
    )


# =============================================================================
# 5. DURABLE SCIENTIFIC GATE
# =============================================================================

banner(
    "STAGE26-4C2 :: DURABLE SCIENTIFIC GATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-4C2 scientific parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before equivalence."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if RUNTIME_ROOT.exists():

    raise RuntimeError(
        "4C2 runtime directory already exists. "
        "Stop for narrow recovery rather than overwriting."
    )


if CHECKPOINT_DIR.exists():

    raise RuntimeError(
        "4C2 Git checkpoint already exists."
    )


# =============================================================================
# 6. FROZEN PROTOCOL / ERRATUM IDENTITY
# =============================================================================

banner(
    "STAGE26-4C2 :: FROZEN IDENTITY GATE"
)


identity_checks = [
    (
        "4C1 protocol",
        PROTOCOL_V1,
        EXPECTED_PROTOCOL_V1_SHA256,
    ),

    (
        "4C1 boundary",
        BOUNDARY_V1,
        EXPECTED_BOUNDARY_V1_SHA256,
    ),

    (
        "worker v2",
        WORKER_V2,
        EXPECTED_WORKER_V2_SHA256,
    ),

    (
        "dependency proof",
        DEPENDENCY_PROOF,
        EXPECTED_PROOF_SHA256,
    ),

    (
        "4C1A erratum",
        ERRATUM,
        EXPECTED_ERRATUM_SHA256,
    ),

    (
        "4C1A receipt",
        ERRATUM_RECEIPT,
        EXPECTED_ERRATUM_RECEIPT_SHA256,
    ),

    (
        "4C1A manifest",
        ERRATUM_MANIFEST,
        EXPECTED_ERRATUM_MANIFEST_SHA256,
    ),

    (
        "historical loader",
        HISTORICAL_LOADER,
        EXPECTED_HISTORICAL_LOADER_SHA256,
    ),
]


for label, path, expected in identity_checks:

    if not path.exists():

        raise FileNotFoundError(
            path
        )

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )

    print(
        f"{label:28s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )

    if not passed:

        raise RuntimeError(
            f"Frozen identity mismatch: {label}"
        )


# =============================================================================
# 7. SCIENTIFIC PERMISSION GATE
# =============================================================================

banner(
    "STAGE26-4C2 :: SCIENTIFIC PERMISSION GATE"
)


protocol = json.loads(
    PROTOCOL_V1.read_text(
        encoding="utf-8"
    )
)

erratum = json.loads(
    ERRATUM.read_text(
        encoding="utf-8"
    )
)


permission_checks = {
    "equivalence_required":
        (
            protocol[
                "mandatory_pre_timing_equivalence_gate"
            ][
                "required"
            ]
            is True
        ),

    "equivalence_start":
        (
            int(
                protocol[
                    "mandatory_pre_timing_equivalence_gate"
                ][
                    "start_index_zero_based"
                ]
            )
            ==
            START_INDEX
        ),

    "equivalence_count":
        (
            int(
                protocol[
                    "mandatory_pre_timing_equivalence_gate"
                ][
                    "flow_count"
                ]
            )
            ==
            EQUIVALENCE_FLOW_COUNT
        ),

    "equivalence_untimed":
        (
            protocol[
                "mandatory_pre_timing_equivalence_gate"
            ][
                "timing_during_equivalence"
            ]
            is False
        ),

    "labels_forbidden":
        (
            protocol[
                "scientific_rules"
            ][
                "labels_may_be_opened"
            ]
            is False
        ),

    "worker_v2_effective":
        (
            erratum[
                "effective_execution_rule"
            ][
                "equivalence_worker"
            ]
            ==
            "stage26_representation_worker_v2.py"
        ),

    "constructor_bypassed":
        (
            erratum[
                "correction"
            ][
                "historical_constructor_bypassed"
            ]
            is True
        ),

    "labels_file_not_opened":
        (
            erratum[
                "correction"
            ][
                "labels_file_opened"
            ]
            is False
        ),

    "timing_still_forbidden":
        (
            erratum[
                "effective_execution_rule"
            ][
                "timing_still_forbidden_until_equivalence_pass_and_git_anchor"
            ]
            is True
        ),
}


for name, passed in permission_checks.items():

    print(
        f"{name:40s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    permission_checks.values()
):

    raise RuntimeError(
        "4C2 scientific permission gate failed."
    )


# =============================================================================
# 8. RELEASE TAR PRESENCE
# =============================================================================

banner(
    "STAGE26-4C2 :: AUTHORITATIVE RELEASE SOURCE"
)


if not MONDAY_TAR.exists():

    raise FileNotFoundError(
        MONDAY_TAR
    )


actual_tar_size = int(
    MONDAY_TAR.stat().st_size
)


print(
    "TAR:"
)

print(
    " ",
    MONDAY_TAR
)

print(
    "Expected bytes:",
    EXPECTED_TAR_BYTES
)

print(
    "Actual bytes  :",
    actual_tar_size
)

print(
    "Frozen SHA256 :",
    EXPECTED_TAR_SHA256
)

print(
    "Full TAR rehash now: NO"
)


if actual_tar_size != EXPECTED_TAR_BYTES:

    raise RuntimeError(
        "Monday Release TAR size mismatch."
    )


# =============================================================================
# 9. RESTORE ONLY THREE REQUIRED RELEASE MEMBERS
# =============================================================================

banner(
    "STAGE26-4C2 :: RESTORE REQUIRED MEMBERS ONLY"
)


RUNTIME_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

CORPUS_RUNTIME_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


restored_members = []


with tarfile.open(
    MONDAY_TAR,
    mode="r:",
) as tf:

    tar_members = {
        member.name:
            member
        for member in tf.getmembers()
        if member.isfile()
    }


    # labels.npy may exist in TAR, but must never be extracted/opened.
    if FORBIDDEN_TAR_MEMBER not in tar_members:

        raise RuntimeError(
            "Expected authoritative labels TAR member not present; "
            "source structure unexpectedly changed."
        )


    for output_name, frozen in REQUIRED_MEMBERS.items():

        tar_member_name = frozen[
            "tar_member"
        ]


        if tar_member_name not in tar_members:

            raise RuntimeError(
                f"Required Release member missing: {tar_member_name}"
            )


        member = tar_members[
            tar_member_name
        ]


        if int(
            member.size
        ) != int(
            frozen[
                "size_bytes"
            ]
        ):

            raise RuntimeError(
                f"TAR member size mismatch: {tar_member_name}"
            )


        source = tf.extractfile(
            member
        )


        if source is None:

            raise RuntimeError(
                f"Could not open TAR member: {tar_member_name}"
            )


        destination = (
            CORPUS_RUNTIME_DIR
            / output_name
        )


        h = hashlib.sha256()

        total = 0


        with destination.open(
            "wb"
        ) as dst:

            while True:

                block = source.read(
                    8 * 1024 * 1024
                )

                if not block:
                    break

                dst.write(
                    block
                )

                h.update(
                    block
                )

                total += len(
                    block
                )


            dst.flush()

            os.fsync(
                dst.fileno()
            )


        digest = h.hexdigest()


        passed = (
            total
            ==
            int(
                frozen[
                    "size_bytes"
                ]
            )
            and
            digest
            ==
            frozen[
                "sha256"
            ]
        )


        print(
            f"{output_name:22s} "
            f"{'PASS' if passed else 'FAIL'} "
            f"{total:12,d} B "
            f"{digest}"
        )


        if not passed:

            raise RuntimeError(
                f"Restored member verification failed: {output_name}"
            )


        restored_members.append(
            {
                "runtime_filename":
                    output_name,

                "source_tar_member":
                    tar_member_name,

                "size_bytes":
                    total,

                "sha256":
                    digest,
            }
        )


# =============================================================================
# 10. PROVE LABEL FILE ABSENCE
# =============================================================================

banner(
    "STAGE26-4C2 :: LABEL-ABSENCE GATE"
)


labels_runtime_path = (
    CORPUS_RUNTIME_DIR
    / FORBIDDEN_MEMBER
)


print(
    "Runtime labels path:"
)

print(
    " ",
    labels_runtime_path
)

print(
    "Exists:",
    labels_runtime_path.exists()
)


if labels_runtime_path.exists():

    raise RuntimeError(
        "labels.npy must not exist in equivalence runtime corpus directory."
    )


actual_runtime_files = sorted(
    path.name
    for path in CORPUS_RUNTIME_DIR.iterdir()
    if path.is_file()
)

expected_runtime_files = sorted(
    REQUIRED_MEMBERS.keys()
)


print(
    "\nExpected runtime files:",
    expected_runtime_files
)

print(
    "Actual runtime files  :",
    actual_runtime_files
)


if actual_runtime_files != expected_runtime_files:

    raise RuntimeError(
        "Unexpected runtime corpus files."
    )


print(
    "\n[PASS] Runtime representation source contains exactly three files."
)

print(
    "labels.npy restored: NO"
)

print(
    "corpus regenerated : NO"
)


# =============================================================================
# 11. WRITE RESTORATION RECEIPT
# =============================================================================

restoration_receipt = {
    "schema":
        "stage26_4c2_release_member_restoration_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "checkpoint":
        "STAGE26-4C2",

    "scientific_parent":
        EXPECTED_PARENT,

    "authoritative_release": {
        "asset":
            "stage20-Monday-compact-corpus-v1.tar",

        "size_bytes":
            EXPECTED_TAR_BYTES,

        "sha256":
            EXPECTED_TAR_SHA256,
    },

    "runtime_directory":
        str(
            CORPUS_RUNTIME_DIR
        ),

    "restored_members":
        restored_members,

    "runtime_file_count":
        len(
            actual_runtime_files
        ),

    "labels_member_present_in_source_tar":
        True,

    "labels_member_extracted":
        False,

    "labels_file_present_in_runtime":
        False,

    "release_members_restored":
        True,

    "release_corpus_regenerated":
        False,

    "pcap_accessed":
        False,

    "timing_performed":
        False,

    "gpu_used":
        False,
}


atomic_json(
    RESTORATION_RECEIPT_PATH,
    restoration_receipt,
)


restoration_receipt_sha = sha256_file(
    RESTORATION_RECEIPT_PATH
)


print(
    "\nRestoration receipt SHA256:"
)

print(
    " ",
    restoration_receipt_sha
)


# =============================================================================
# 12. WRITE FROZEN EQUIVALENCE CONFIG
# =============================================================================

banner(
    "STAGE26-4C2 :: WRITE EQUIVALENCE CONFIG"
)


config = {
    "schema":
        "stage26_4c2_representation_equivalence_config_v1",

    "mode":
        "VALIDATE_EQUIVALENCE",

    "scientific_parent":
        EXPECTED_PARENT,

    "worker_sha256":
        EXPECTED_WORKER_V2_SHA256,

    "corpus_dir":
        str(
            CORPUS_RUNTIME_DIR
        ),

    "expected_flow_count":
        FLOW_COUNT,

    "historical_loader_path":
        str(
            HISTORICAL_LOADER
        ),

    "historical_loader_sha256":
        EXPECTED_HISTORICAL_LOADER_SHA256,

    "historical_loader_class":
        HISTORICAL_LOADER_CLASS,

    "historical_reconstruct_source_sha256":
        EXPECTED_RECONSTRUCT_SOURCE_SHA256,

    "start_index":
        START_INDEX,

    "flow_count":
        EQUIVALENCE_FLOW_COUNT,

    "end_index_exclusive":
        END_INDEX_EXCLUSIVE,

    "affinity":
        CPU_AFFINITY,

    "result_path":
        str(
            RESULT_PATH
        ),

    "timing_allowed":
        False,

    "labels_allowed":
        False,

    "labels_file_present":
        False,

    "gpu_allowed":
        False,
}


atomic_json(
    CONFIG_PATH,
    config,
)


config_sha = sha256_file(
    CONFIG_PATH
)


print(
    "Config SHA256:"
)

print(
    " ",
    config_sha
)


# =============================================================================
# 13. RUN FROZEN UNTIMED EQUIVALENCE
# =============================================================================

banner(
    "STAGE26-4C2 :: RUN 128-FLOW UNTIMED EQUIVALENCE"
)


if RESULT_PATH.exists():

    raise RuntimeError(
        "Equivalence result already exists."
    )


worker_env = os.environ.copy()


# Thread discipline only; equivalence is not performance measurement.
for key in [
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
]:

    worker_env[
        key
    ] = "1"


# Ensure no GPU visibility is introduced.
worker_env[
    "CUDA_VISIBLE_DEVICES"
] = ""


print(
    "Mode        : VALIDATE_EQUIVALENCE"
)

print(
    "Flows       :",
    f"[{START_INDEX}, {END_INDEX_EXCLUSIVE})"
)

print(
    "Count       :",
    EQUIVALENCE_FLOW_COUNT
)

print(
    "Affinity    :",
    CPU_AFFINITY
)

print(
    "Timing      : DISABLED"
)

print(
    "Labels file : ABSENT"
)

print(
    "GPU         : DISABLED"
)


worker_run = run(
    [
        sys.executable,
        str(
            WORKER_V2
        ),
        str(
            CONFIG_PATH
        ),
    ],
    cwd=REPO,
    env=worker_env,
    check=False,
    text=True,
    timeout=600,
)


print(
    "\nWorker return code:",
    worker_run.returncode
)


if worker_run.stdout.strip():

    print(
        "\nWorker stdout/stderr:"
    )

    print(
        worker_run.stdout
    )


if worker_run.returncode != 0:

    raise RuntimeError(
        "Frozen Stage26-4C2 equivalence worker failed. "
        "Do NOT alter protocol; perform narrow diagnosis."
    )


if not RESULT_PATH.exists():

    raise RuntimeError(
        "Worker exited successfully but produced no result."
    )


result_sha = sha256_file(
    RESULT_PATH
)

result = json.loads(
    RESULT_PATH.read_text(
        encoding="utf-8"
    )
)


# =============================================================================
# 14. STRICT RESULT AUDIT
# =============================================================================

banner(
    "STAGE26-4C2 :: EQUIVALENCE RESULT AUDIT"
)


result_checks = {
    "status_PASS":
        (
            result[
                "status"
            ]
            ==
            "PASS"
        ),

    "mode_equivalence":
        (
            result[
                "mode"
            ]
            ==
            "VALIDATE_EQUIVALENCE"
        ),

    "start_exact":
        (
            int(
                result[
                    "start_index"
                ]
            )
            ==
            START_INDEX
        ),

    "count_exact":
        (
            int(
                result[
                    "flow_count"
                ]
            )
            ==
            EQUIVALENCE_FLOW_COUNT
        ),

    "end_exact":
        (
            int(
                result[
                    "end_index_exclusive"
                ]
            )
            ==
            END_INDEX_EXCLUSIVE
        ),

    "population_exact":
        (
            int(
                result[
                    "population_flow_count"
                ]
            )
            ==
            FLOW_COUNT
        ),

    "image_shape_exact":
        (
            result[
                "image_shape_per_flow"
            ]
            ==
            [
                64,
                256,
            ]
        ),

    "image_dtype_exact":
        (
            result[
                "image_dtype"
            ]
            ==
            "uint8"
        ),

    "mask_shape_exact":
        (
            result[
                "padding_mask_shape_per_flow"
            ]
            ==
            [
                64,
                256,
            ]
        ),

    "mask_dtype_exact":
        (
            result[
                "padding_mask_dtype"
            ]
            ==
            "bool"
        ),

    "timing_false":
        (
            result[
                "timing_performed"
            ]
            is False
        ),

    "historical_constructor_false":
        (
            result[
                "historical_constructor_invoked"
            ]
            is False
        ),

    "historical_labels_file_false":
        (
            result[
                "historical_labels_file_opened"
            ]
            is False
        ),

    "label_shim_true":
        (
            result[
                "label_neutral_validation_shim_used"
            ]
            is True
        ),

    "outputs_compared_exact":
        (
            result[
                "historical_outputs_compared"
            ]
            ==
            [
                "image",
                "padding_mask",
            ]
        ),

    "label_output_not_compared":
        (
            result[
                "historical_label_output_compared"
            ]
            is False
        ),

    "labels_accessed_false":
        (
            result[
                "labels_accessed"
            ]
            is False
        ),

    "pcap_false":
        (
            result[
                "pcap_accessed"
            ]
            is False
        ),

    "corpus_created_false":
        (
            result[
                "corpus_created_or_modified"
            ]
            is False
        ),

    "gpu_false":
        (
            result[
                "gpu_used"
            ]
            is False
        ),
}


for name, passed in result_checks.items():

    print(
        f"{name:44s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    result_checks.values()
):

    raise RuntimeError(
        "Stage26-4C2 equivalence result audit failed."
    )


fingerprint = result[
    "representation_fingerprint_sha256"
]


if (
    not isinstance(
        fingerprint,
        str,
    )
    or
    len(
        fingerprint
    )
    != 64
):

    raise RuntimeError(
        "Invalid equivalence representation fingerprint."
    )


print(
    "\nRepresentation equivalence fingerprint:"
)

print(
    " ",
    fingerprint
)

print(
    "\nResult SHA256:"
)

print(
    " ",
    result_sha
)


# Re-prove labels.npy remained absent AFTER worker execution.
if labels_runtime_path.exists():

    raise RuntimeError(
        "labels.npy appeared in runtime after equivalence."
    )


print(
    "\nlabels.npy after worker: ABSENT"
)


# =============================================================================
# 15. WRITE EQUIVALENCE RECEIPT
# =============================================================================

banner(
    "STAGE26-4C2 :: WRITE EQUIVALENCE RECEIPT"
)


equivalence_receipt = {
    "schema":
        "stage26_4c2_representation_equivalence_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4C2",

    "status":
        "PASS_EXACT_128_FLOW_REPRESENTATION_EQUIVALENCE",

    "scientific_parent":
        EXPECTED_PARENT,

    "frozen_inputs": {
        "protocol_sha256":
            EXPECTED_PROTOCOL_V1_SHA256,

        "boundary_sha256":
            EXPECTED_BOUNDARY_V1_SHA256,

        "worker_v2_sha256":
            EXPECTED_WORKER_V2_SHA256,

        "erratum_sha256":
            EXPECTED_ERRATUM_SHA256,

        "historical_loader_sha256":
            EXPECTED_HISTORICAL_LOADER_SHA256,

        "historical_reconstruct_source_sha256":
            EXPECTED_RECONSTRUCT_SOURCE_SHA256,

        "release_tar_sha256":
            EXPECTED_TAR_SHA256,
    },

    "restoration": {
        "receipt_sha256":
            restoration_receipt_sha,

        "member_count":
            3,

        "labels_member_extracted":
            False,

        "release_corpus_regenerated":
            False,
    },

    "equivalence_config": {
        "sha256":
            config_sha,

        "start_index_zero_based":
            START_INDEX,

        "flow_count":
            EQUIVALENCE_FLOW_COUNT,

        "end_index_exclusive":
            END_INDEX_EXCLUSIVE,

        "CPU_affinity":
            CPU_AFFINITY,

        "timed":
            False,
    },

    "equivalence_result": {
        "sha256":
            result_sha,

        "representation_fingerprint_sha256":
            fingerprint,

        "image_shape":
            [
                64,
                256,
            ],

        "image_dtype":
            "uint8",

        "padding_mask_shape":
            [
                64,
                256,
            ],

        "padding_mask_dtype":
            "bool",

        "all_128_images_exact":
            True,

        "all_128_padding_masks_exact":
            True,
    },

    "label_free_validation": {
        "historical_constructor_invoked":
            False,

        "historical_labels_file_opened":
            False,

        "runtime_labels_file_present":
            False,

        "label_neutral_validation_shim_used":
            True,

        "historical_label_output_compared":
            False,
    },

    "scientific_state": {
        "release_members_restored":
            True,

        "release_corpus_regenerated":
            False,

        "dense_representation_materialized_transiently_for_equivalence":
            True,

        "dense_representation_persisted":
            False,

        "equivalence_executed":
            True,

        "representation_timing_performed":
            False,

        "performance_observations_created":
            False,

        "labels_accessed":
            False,

        "pcap_accessed":
            False,

        "models_loaded":
            False,

        "inference_performed":
            False,

        "Thursday_accessed":
            False,

        "Friday_accessed":
            False,

        "gpu_used":
            False,
    },

    "timing_permission_after_git_anchor":
        True,

    "timing_permission_condition":
        (
            "Representation timing becomes permitted only after this exact "
            "equivalence checkpoint is committed, pushed, and remotely "
            "byte/scientifically verified."
        ),
}


atomic_json(
    EQUIVALENCE_RECEIPT_RUNTIME,
    equivalence_receipt,
)


equivalence_receipt_sha = sha256_file(
    EQUIVALENCE_RECEIPT_RUNTIME
)


print(
    "Equivalence receipt SHA256:"
)

print(
    " ",
    equivalence_receipt_sha
)


# =============================================================================
# 16. CREATE DURABLE GIT CHECKPOINT
# =============================================================================

banner(
    "STAGE26-4C2 :: CREATE DURABLE CHECKPOINT"
)


CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


for source, destination in [
    (
        CONFIG_PATH,
        CHECKPOINT_CONFIG,
    ),
    (
        RESULT_PATH,
        CHECKPOINT_RESULT,
    ),
    (
        RESTORATION_RECEIPT_PATH,
        CHECKPOINT_RESTORATION,
    ),
    (
        EQUIVALENCE_RECEIPT_RUNTIME,
        CHECKPOINT_RECEIPT,
    ),
]:

    shutil.copy2(
        source,
        destination,
    )


# Byte-exact copy gate.
for source, destination in [
    (
        CONFIG_PATH,
        CHECKPOINT_CONFIG,
    ),
    (
        RESULT_PATH,
        CHECKPOINT_RESULT,
    ),
    (
        RESTORATION_RECEIPT_PATH,
        CHECKPOINT_RESTORATION,
    ),
    (
        EQUIVALENCE_RECEIPT_RUNTIME,
        CHECKPOINT_RECEIPT,
    ),
]:

    if sha256_file(
        source
    ) != sha256_file(
        destination
    ):

        raise RuntimeError(
            f"Checkpoint copy mismatch: {source.name}"
        )


# =============================================================================
# 17. BUILD CHECKPOINT MANIFEST
# =============================================================================

checkpoint_files = [
    CHECKPOINT_CONFIG,
    CHECKPOINT_RESULT,
    CHECKPOINT_RESTORATION,
    CHECKPOINT_RECEIPT,
]


manifest_rows = []


for path in checkpoint_files:

    manifest_rows.append(
        {
            "repo_relative_path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


checkpoint_manifest = {
    "schema":
        "stage26_4c2_representation_equivalence_manifest_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        "READY_FOR_GIT_ANCHOR",

    "parent_commit":
        EXPECTED_PARENT,

    "commit_subject":
        COMMIT_SUBJECT,

    "protocol_sha256":
        EXPECTED_PROTOCOL_V1_SHA256,

    "boundary_sha256":
        EXPECTED_BOUNDARY_V1_SHA256,

    "worker_v2_sha256":
        EXPECTED_WORKER_V2_SHA256,

    "erratum_sha256":
        EXPECTED_ERRATUM_SHA256,

    "historical_loader_sha256":
        EXPECTED_HISTORICAL_LOADER_SHA256,

    "restoration_receipt_sha256":
        restoration_receipt_sha,

    "equivalence_config_sha256":
        config_sha,

    "equivalence_result_sha256":
        result_sha,

    "equivalence_receipt_sha256":
        equivalence_receipt_sha,

    "representation_fingerprint_sha256":
        fingerprint,

    "file_count_excluding_manifest":
        len(
            manifest_rows
        ),

    "files":
        manifest_rows,

    "scientific_boundaries": {
        "equivalence_executed":
            True,

        "equivalence_flow_count":
            EQUIVALENCE_FLOW_COUNT,

        "equivalence_exact":
            True,

        "equivalence_timed":
            False,

        "representation_timing_performed":
            False,

        "labels_accessed":
            False,

        "release_corpus_regenerated":
            False,

        "pcap_accessed":
            False,

        "models_loaded":
            False,

        "gpu_used":
            False,

        "timing_allowed_only_after_remote_anchor":
            True,
    },
}


atomic_json(
    CHECKPOINT_MANIFEST,
    checkpoint_manifest,
)


checkpoint_manifest_sha = sha256_file(
    CHECKPOINT_MANIFEST
)


print(
    "Config      :",
    config_sha
)

print(
    "Result      :",
    result_sha
)

print(
    "Restoration :",
    restoration_receipt_sha
)

print(
    "Receipt     :",
    equivalence_receipt_sha
)

print(
    "Manifest    :",
    checkpoint_manifest_sha
)

print(
    "Fingerprint :",
    fingerprint
)


# =============================================================================
# 18. LOCAL CHECKPOINT AUDIT
# =============================================================================

banner(
    "STAGE26-4C2 :: LOCAL CHECKPOINT AUDIT"
)


for row in manifest_rows:

    path = (
        REPO
        / row[
            "repo_relative_path"
        ]
    )

    size = int(
        path.stat().st_size
    )

    digest = sha256_file(
        path
    )

    passed = (
        size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        digest
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{size:10,d} B "
        f"{digest} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Local 4C2 checkpoint audit failed."
        )


# =============================================================================
# 19. GIT CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-4C2 :: GIT CHANGE AUDIT"
)


repo_status = git(
    "status",
    "--porcelain",
)


print(
    repo_status
)


if not repo_status:

    raise RuntimeError(
        "Expected uncommitted 4C2 checkpoint."
    )


unexpected = []


for line in repo_status.splitlines():

    rel = line[
        3:
    ]


    if not rel.startswith(
        str(
            CHECKPOINT_REL
        )
        +
        "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository changes:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 20. GIT IDENTITY
# =============================================================================

author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_PARENT,
)

author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_PARENT,
)


if (
    not author_name.strip()
    or
    "@"
    not in
    author_email
):

    raise RuntimeError(
        "Could not resolve Git author identity."
    )


git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


print(
    "\nGit author:"
)

print(
    " ",
    author_name,
    "<" + author_email + ">"
)


# =============================================================================
# 21. COMMIT
# =============================================================================

banner(
    "STAGE26-4C2 :: COMMIT"
)


git(
    "add",
    str(
        CHECKPOINT_REL
    ),
)


print(
    git(
        "diff",
        "--cached",
        "--name-status",
    )
)


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "4C2 commit parent mismatch."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "4C2 commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository not clean after 4C2 commit."
    )


# =============================================================================
# 22. PUSH
# =============================================================================

banner(
    "STAGE26-4C2 :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    pushed = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
        check=True,
        text=True,
    )


    print(
        pushed.stdout.strip()
    )


github_token = None


# =============================================================================
# 23. REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-4C2 :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "origin/main != new 4C2 commit."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote 4C2 parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote 4C2 subject mismatch."
    )


# =============================================================================
# 24. REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-4C2 :: REMOTE BYTE VERIFICATION"
)


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    str(
        CHECKPOINT_MANIFEST.relative_to(
            REPO
        )
    ),
)

remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "Remote manifest SHA256:"
)

print(
    " ",
    remote_manifest_sha
)


if remote_manifest_sha != checkpoint_manifest_sha:

    raise RuntimeError(
        "Remote 4C2 manifest mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


if (
    int(
        remote_manifest[
            "file_count_excluding_manifest"
        ]
    )
    !=
    4
):

    raise RuntimeError(
        "Unexpected 4C2 remote file count."
    )


for row in remote_manifest[
    "files"
]:

    data = git_blob_bytes(
        "origin/main",
        row[
            "repo_relative_path"
        ],
    )

    actual_size = len(
        data
    )

    actual_sha = sha256_bytes(
        data
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Remote 4C2 byte verification failed."
        )


# =============================================================================
# 25. REMOTE SCIENTIFIC AUDIT
# =============================================================================

banner(
    "STAGE26-4C2 :: REMOTE SCIENTIFIC AUDIT"
)


remote_result = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            CHECKPOINT_RESULT.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)


remote_receipt = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            CHECKPOINT_RECEIPT.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)


remote_restoration = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            CHECKPOINT_RESTORATION.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)


remote_checks = {
    "equivalence_PASS":
        (
            remote_result[
                "status"
            ]
            ==
            "PASS"
        ),

    "mode_exact":
        (
            remote_result[
                "mode"
            ]
            ==
            "VALIDATE_EQUIVALENCE"
        ),

    "128_flows":
        (
            int(
                remote_result[
                    "flow_count"
                ]
            )
            ==
            128
        ),

    "fingerprint_exact":
        (
            remote_result[
                "representation_fingerprint_sha256"
            ]
            ==
            fingerprint
        ),

    "timing_false":
        (
            remote_result[
                "timing_performed"
            ]
            is False
        ),

    "constructor_false":
        (
            remote_result[
                "historical_constructor_invoked"
            ]
            is False
        ),

    "labels_file_false":
        (
            remote_result[
                "historical_labels_file_opened"
            ]
            is False
        ),

    "label_shim_true":
        (
            remote_result[
                "label_neutral_validation_shim_used"
            ]
            is True
        ),

    "labels_not_restored":
        (
            remote_restoration[
                "labels_member_extracted"
            ]
            is False
        ),

    "corpus_not_regenerated":
        (
            remote_restoration[
                "release_corpus_regenerated"
            ]
            is False
        ),

    "receipt_PASS":
        (
            remote_receipt[
                "status"
            ]
            ==
            "PASS_EXACT_128_FLOW_REPRESENTATION_EQUIVALENCE"
        ),

    "all_images_exact":
        (
            remote_receipt[
                "equivalence_result"
            ][
                "all_128_images_exact"
            ]
            is True
        ),

    "all_masks_exact":
        (
            remote_receipt[
                "equivalence_result"
            ][
                "all_128_padding_masks_exact"
            ]
            is True
        ),

    "representation_timing_false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "representation_timing_performed"
            ]
            is False
        ),

    "performance_observations_false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "performance_observations_created"
            ]
            is False
        ),

    "timing_permission_true":
        (
            remote_receipt[
                "timing_permission_after_git_anchor"
            ]
            is True
        ),

    "gpu_false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "gpu_used"
            ]
            is False
        ),
}


for name, passed in remote_checks.items():

    print(
        f"{name:44s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    remote_checks.values()
):

    raise RuntimeError(
        "Remote Stage26-4C2 scientific audit failed."
    )


# =============================================================================
# 26. FINAL GIT AUDIT
# =============================================================================

banner(
    "STAGE26-4C2 :: FINAL GIT AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != final_remote:

    raise RuntimeError(
        "Local/remote divergence after 4C2."
    )


if final_status:

    raise RuntimeError(
        "Repository not clean after 4C2."
    )


# =============================================================================
# 27. CLOSURE
# =============================================================================

banner(
    "STAGE26-4C2 REPRESENTATION EQUIVALENCE GATE COMPLETE"
)


print(
    "NEW DURABLE COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nPARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nAUTHORITATIVE RELEASE RESTORATION:"
)

print(
    "  encoded_bytes.bin   : VERIFIED"
)

print(
    "  flow_offsets.npy    : VERIFIED"
)

print(
    "  packet_lengths.npy  : VERIFIED"
)

print(
    "  labels.npy          : NOT EXTRACTED"
)

print(
    "  corpus regenerated  : NO"
)


print(
    "\nUNTIMED EQUIVALENCE:"
)

print(
    "  range       :",
    f"[{START_INDEX}, {END_INDEX_EXCLUSIVE})"
)

print(
    "  flows       :",
    EQUIVALENCE_FLOW_COUNT
)

print(
    "  image       : exact uint8 [64,256]"
)

print(
    "  padding mask: exact bool [64,256]"
)

print(
    "  label       : not compared"
)

print(
    "  historical constructor invoked : NO"
)

print(
    "  labels file opened             : NO"
)

print(
    "  timing                         : NO"
)


print(
    "\nREPRESENTATION FINGERPRINT:"
)

print(
    " ",
    fingerprint
)


print(
    "\nCHECKPOINT HASHES:"
)

print(
    "  config      :",
    config_sha
)

print(
    "  result      :",
    result_sha
)

print(
    "  restoration :",
    restoration_receipt_sha
)

print(
    "  receipt     :",
    equivalence_receipt_sha
)

print(
    "  manifest    :",
    checkpoint_manifest_sha
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  commit                   : PASS"
)

print(
    "  parent                   : PASS"
)

print(
    "  manifest                 : PASS"
)

print(
    "  four checkpoint files    : PASS"
)

print(
    "  scientific content       : PASS"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  REPRESENTATION EQUIVALENCE PASSED       : YES"
)

print(
    "  EQUIVALENCE REMOTELY ANCHORED           : YES"
)

print(
    "  TRANSIENT REPRESENTATIONS MATERIALIZED  : YES — validation only"
)

print(
    "  DENSE REPRESENTATIONS PERSISTED         : NO"
)

print(
    "  REPRESENTATION PERFORMANCE MEASURED     : NO"
)

print(
    "  LABELS ACCESSED                         : NO"
)

print(
    "  PCAP ACCESSED                           : NO"
)

print(
    "  RELEASE CORPUS REGENERATED              : NO"
)

print(
    "  MODELS LOADED                           : NO"
)

print(
    "  GPU                                     : NO"
)


print(
    "\nTIMING PERMISSION:"
)

print(
    "  Stage26-4C representation timing is NOW PERMITTED,"
)

print(
    "  because the frozen equivalence gate has passed and"
)

print(
    "  has been remotely Git-anchored."
)


print(
    "\nNEXT:"
)

print(
    "  Execute the already-frozen CPU1 representation timing"
)

print(
    "  conditions for B=[1,64,256,1024,8192]."
)

print(
    "  Do not change sample, warmups, timed iterations,"
)

print(
    "  affinity, thread count, or output boundary."
)


STAGE26-4C2 :: DURABLE SCIENTIFIC GATE
Expected parent: 6a55aca543111e36e9fe83f16f008932d2392a13
Local HEAD     : 6a55aca543111e36e9fe83f16f008932d2392a13
origin/main    : 6a55aca543111e36e9fe83f16f008932d2392a13
Repo clean     : True

STAGE26-4C2 :: FROZEN IDENTITY GATE
4C1 protocol                 PASS 4c446212f1af49a6c70c3ef6fcd22facb3f67c650ce6f12a4790807bd58bcb2f
4C1 boundary                 PASS 88efc94664d9658b1e0601506ac4ea369300a98a05049e79280a77107c0828ba
worker v2                    PASS ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23
dependency proof             PASS 05ecedbb0d973173c276e5bcec54041538b345614c296561867622933781b597
4C1A erratum                 PASS 1f6748b3042b64626de73c74f9a9ff2f8c1f9ce5e6b445a5dd2030137f951839
4C1A receipt                 PASS 3341f747f95b9da36c17c39727aaa139a71858551c2cadc31c233454a98f8ddb
4C1A manifest                PASS b3dd2844829930f016f760805d75f066795ecb9f86b4b52a668dee4beea2470e
historical loader            PASS 

In [9]:
# =============================================================================
# STAGE26-4C3
# FROZEN CPU1 COMPACT-CORPUS REPRESENTATION TIMING
# COMPLETE ALL FIVE CONDITIONS -> COMMIT -> PUSH -> REMOTE VERIFY
#
# DURABLE SCIENTIFIC PARENT:
#   d6ec86b1a6fb7ce69c254dd1c6238eb3ec0f459b
#
# EFFECTIVE REPRESENTATION WORKER:
#   stage26_representation_worker_v2.py
#
# WORKER SHA256:
#   ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23
#
# PASSED / REMOTELY ANCHORED EQUIVALENCE:
#   128 exact flows
#   fingerprint:
#   1f64ea3719abcdd658d2a16f6c59348f0465cb800cc08d0f2fc697d14f31b185
#
# ---------------------------------------------------------------------------
# FROZEN REPRESENTATION COMPONENT
# ---------------------------------------------------------------------------
#
# Existing Stage20 compact Release corpus:
#
#   encoded_bytes.bin
#   flow_offsets.npy
#   packet_lengths.npy
#
#             |
#             v
#
#   dense image         uint8 [B,64,256]
#   byte padding mask   bool  [B,64,256]
#
# ---------------------------------------------------------------------------
# FROZEN CPU1 CONDITIONS
# ---------------------------------------------------------------------------
#
# CPU affinity: [0]
# thread count: 1
#
# Sample start: 26042
#
# B=1       warm=50  timed=200
# B=64      warm=30  timed=150
# B=256     warm=20  timed=100
# B=1024    warm=10  timed=50
# B=8192    warm=5   timed=20
#
# ONE FRESH PROCESS PER BATCH CONDITION.
#
# ---------------------------------------------------------------------------
# TIMER INCLUDES
# ---------------------------------------------------------------------------
#
#   - memory-mapped offset/length reads
#   - memory-mapped encoded-byte reads
#   - allocation of dense uint8 output batch
#   - allocation of dense bool mask batch
#   - flow-by-flow reconstruction
#   - copies into batch outputs
#
# TIMER EXCLUDES
# ---------------------------------------------------------------------------
#
#   - Python process startup
#   - RepresentationSource initialization
#   - Release TAR extraction
#   - output fingerprinting
#   - JSON serialization
#   - float32 conversion
#   - /255 scaling
#   - model inference
#
# ---------------------------------------------------------------------------
# CACHE SEMANTICS
# ---------------------------------------------------------------------------
#
#   - exact batch is warmed before timed iterations
#   - OS page cache is NOT flushed/manipulated
#   - this is NOT a cold-storage benchmark
#
# IMPORTANT:
#   DO NOT rehash encoded_bytes.bin before timing.
#
#   Its exact SHA256 was verified during Stage26-4C2 and remotely anchored.
#   Re-reading all 522 MB here would unnecessarily alter page-cache state.
#
# ---------------------------------------------------------------------------
# SCIENTIFIC EXCLUSIONS
# ---------------------------------------------------------------------------
#
#   NO labels.npy
#   NO PCAP
#   NO corpus regeneration
#   NO raw packet masking/encoding
#   NO float32 / 255
#   NO models
#   NO inference
#   NO GPU
#
# ---------------------------------------------------------------------------
# SUMMARY RULE
# ---------------------------------------------------------------------------
#
# Latency quantiles use NumPy percentile(method="linear").
#
# p50 : all conditions
# p95 : all conditions
# p99 : ONLY where timed iteration count >= 100
#
# Therefore:
#   B1, B64, B256  -> p99 reported
#   B1024, B8192   -> p99 NOT reported
#
# This is a reporting implementation of the already-frozen summary metrics,
# not a change to the scientific measurement protocol.
# =============================================================================

from __future__ import annotations

import os
import sys
import csv
import json
import math
import stat
import time
import shutil
import hashlib
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import psutil


# =============================================================================
# 0. DURABLE IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

EXPECTED_PARENT = (
    "d6ec86b1a6fb7ce69c254dd1c6238eb3ec0f459b"
)

COMMIT_SUBJECT = (
    "stage26: anchor CPU1 representation timing"
)


# -----------------------------------------------------------------------------
# Stage26-4C1 frozen protocol
# -----------------------------------------------------------------------------

LOCK_4C1 = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c_representation_protocol_lock"
)

PROTOCOL_PATH = (
    LOCK_4C1
    / "stage26_representation_protocol.json"
)

BOUNDARY_PATH = (
    LOCK_4C1
    / "stage26_representation_boundary_map.json"
)

EXPECTED_PROTOCOL_SHA256 = (
    "4c446212f1af49a6c70c3ef6fcd22facb3f67c650ce6f12a4790807bd58bcb2f"
)

EXPECTED_BOUNDARY_SHA256 = (
    "88efc94664d9658b1e0601506ac4ea369300a98a05049e79280a77107c0828ba"
)


# -----------------------------------------------------------------------------
# Stage26-4C1A effective worker / erratum
# -----------------------------------------------------------------------------

ERRATUM_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c1a_label_free_equivalence_erratum"
)

WORKER = (
    ERRATUM_DIR
    / "stage26_representation_worker_v2.py"
)

ERRATUM_PATH = (
    ERRATUM_DIR
    / "stage26_4c1a_equivalence_erratum.json"
)

EXPECTED_WORKER_SHA256 = (
    "ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23"
)

EXPECTED_ERRATUM_SHA256 = (
    "1f6748b3042b64626de73c74f9a9ff2f8c1f9ce5e6b445a5dd2030137f951839"
)


# -----------------------------------------------------------------------------
# Stage26-4C2 anchored equivalence
# -----------------------------------------------------------------------------

EQUIVALENCE_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c2_representation_equivalence"
)

EQUIVALENCE_RESULT = (
    EQUIVALENCE_DIR
    / "stage26_4c2_equivalence_result.json"
)

EQUIVALENCE_RECEIPT = (
    EQUIVALENCE_DIR
    / "stage26_4c2_equivalence_receipt.json"
)

EQUIVALENCE_RESTORATION = (
    EQUIVALENCE_DIR
    / "stage26_4c2_release_member_restoration_receipt.json"
)

EQUIVALENCE_MANIFEST = (
    EQUIVALENCE_DIR
    / "stage26_4c2_representation_equivalence_manifest.json"
)

EXPECTED_EQ_RESULT_SHA256 = (
    "d8b28e8aea4f898bf4c6d5eba95706bbffeaca4270e596ee63529d013e88da10"
)

EXPECTED_EQ_RECEIPT_SHA256 = (
    "4dd18384f0c5d2dbc68711fbb4fd11dc7facd550a4be9db2b3ab13cc5fda2351"
)

EXPECTED_EQ_RESTORATION_SHA256 = (
    "6f31d78553a24695eb378abd65823dfaa400325557e78c1d26686d031fb8a3d2"
)

EXPECTED_EQ_MANIFEST_SHA256 = (
    "814914ddc47b824ea8d54d4732ab2ffdbd0b3847d60ef7a69a993e4298281121"
)

EXPECTED_EQ_FINGERPRINT = (
    "1f64ea3719abcdd658d2a16f6c59348f0465cb800cc08d0f2fc697d14f31b185"
)


# =============================================================================
# 1. AUTHORITATIVE RUNTIME REPRESENTATION SOURCE
# =============================================================================

CORPUS_RUNTIME_DIR = (
    STAGE26_ROOT
    / "representation"
    / "stage26_4c2_equivalence"
    / "Monday_release_representation_subset"
)

ENCODED_PATH = (
    CORPUS_RUNTIME_DIR
    / "encoded_bytes.bin"
)

OFFSETS_PATH = (
    CORPUS_RUNTIME_DIR
    / "flow_offsets.npy"
)

LENGTHS_PATH = (
    CORPUS_RUNTIME_DIR
    / "packet_lengths.npy"
)

LABELS_PATH = (
    CORPUS_RUNTIME_DIR
    / "labels.npy"
)


SOURCE_IDENTITIES = {
    "encoded_bytes.bin": {
        "path":
            ENCODED_PATH,

        "size_bytes":
            522_845_159,

        "sha256":
            "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",
    },

    "flow_offsets.npy": {
        "path":
            OFFSETS_PATH,

        "size_bytes":
            4_228_208,

        "sha256":
            "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",
    },

    "packet_lengths.npy": {
        "path":
            LENGTHS_PATH,

        "size_bytes":
            67_649_280,

        "sha256":
            "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
    },
}


FLOW_COUNT = (
    528_509
)

ROWS = (
    64
)

COLS = (
    256
)


# =============================================================================
# 2. FROZEN CPU1 TIMING PLAN
# =============================================================================

SAMPLE_START = (
    26_042
)

BATCH_SIZES = [
    1,
    64,
    256,
    1024,
    8192,
]

CPU_AFFINITY = [
    0
]

THREAD_COUNT = (
    1
)

CONDITION_TIMEOUT_SECONDS = (
    600
)


ITERATION_POLICY = {
    1: {
        "warmup_runs":
            50,

        "timed_runs":
            200,
    },

    64: {
        "warmup_runs":
            30,

        "timed_runs":
            150,
    },

    256: {
        "warmup_runs":
            20,

        "timed_runs":
            100,
    },

    1024: {
        "warmup_runs":
            10,

        "timed_runs":
            50,
    },

    8192: {
        "warmup_runs":
            5,

        "timed_runs":
            20,
    },
}


ENVIRONMENT_GATE = {
    "cpu_utilization_percent_max":
        20.0,

    "available_ram_gib_min":
        8.0,

    "cpu_utilization_sampling_seconds":
        1.0,

    "retry_count":
        3,

    "retry_cooldown_seconds":
        5,
}


# =============================================================================
# 3. RUNTIME OUTPUTS
# =============================================================================

RUNTIME_ROOT = (
    STAGE26_ROOT
    / "representation"
    / "stage26_4c3_cpu1_timing"
)

RUNTIME_CONDITIONS = (
    RUNTIME_ROOT
    / "conditions"
)

EXECUTION_PLAN_PATH = (
    RUNTIME_ROOT
    / "stage26_4c3_execution_plan.json"
)

RAW_CSV_PATH = (
    RUNTIME_ROOT
    / "stage26_4c3_raw_observations.csv"
)

SUMMARY_JSON_PATH = (
    RUNTIME_ROOT
    / "stage26_4c3_cpu1_representation_summary.json"
)

SUMMARY_CSV_PATH = (
    RUNTIME_ROOT
    / "stage26_4c3_cpu1_representation_summary.csv"
)

RECEIPT_RUNTIME_PATH = (
    RUNTIME_ROOT
    / "stage26_4c3_cpu1_representation_timing_receipt.json"
)


# =============================================================================
# 4. DURABLE CHECKPOINT OUTPUTS
# =============================================================================

CHECKPOINT_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_4c3_cpu1_representation_timing"
)

CHECKPOINT_DIR = (
    REPO
    / CHECKPOINT_REL
)

CHECKPOINT_CONDITIONS = (
    CHECKPOINT_DIR
    / "conditions"
)

CHECKPOINT_EXECUTION_PLAN = (
    CHECKPOINT_DIR
    / EXECUTION_PLAN_PATH.name
)

CHECKPOINT_RAW_CSV = (
    CHECKPOINT_DIR
    / RAW_CSV_PATH.name
)

CHECKPOINT_SUMMARY_JSON = (
    CHECKPOINT_DIR
    / SUMMARY_JSON_PATH.name
)

CHECKPOINT_SUMMARY_CSV = (
    CHECKPOINT_DIR
    / SUMMARY_CSV_PATH.name
)

CHECKPOINT_RECEIPT = (
    CHECKPOINT_DIR
    / RECEIPT_RUNTIME_PATH.name
)

CHECKPOINT_MANIFEST = (
    CHECKPOINT_DIR
    / "stage26_4c3_cpu1_representation_timing_manifest.json"
)


# =============================================================================
# 5. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 122
    )

    print(text)

    print(
        "=" * 122
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
    timeout=None,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
        timeout=timeout,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        check=True,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        check=True,
        text=False,
    )

    return bytes(
        p.stdout
    )


def percentile(
    values,
    q,
):

    values = np.asarray(
        values,
        dtype=np.float64,
    )

    return float(
        np.percentile(
            values,
            q,
            method="linear",
        )
    )


def environment_gate():

    attempts = []


    for attempt in range(
        1,
        ENVIRONMENT_GATE[
            "retry_count"
        ]
        +
        1,
    ):

        cpu_percent = float(
            psutil.cpu_percent(
                interval=ENVIRONMENT_GATE[
                    "cpu_utilization_sampling_seconds"
                ]
            )
        )

        available_ram_gib = float(
            psutil.virtual_memory().available
            /
            (
                1024
                **
                3
            )
        )

        passed = (
            cpu_percent
            <=
            ENVIRONMENT_GATE[
                "cpu_utilization_percent_max"
            ]
            and
            available_ram_gib
            >=
            ENVIRONMENT_GATE[
                "available_ram_gib_min"
            ]
        )


        record = {
            "attempt":
                attempt,

            "cpu_utilization_percent":
                cpu_percent,

            "available_ram_gib":
                available_ram_gib,

            "passed":
                passed,
        }


        attempts.append(
            record
        )


        print(
            f"  gate attempt {attempt}: "
            f"CPU={cpu_percent:.1f}% "
            f"RAM={available_ram_gib:.3f} GiB "
            f"{'PASS' if passed else 'FAIL'}"
        )


        if passed:

            return {
                "status":
                    "PASS",

                "successful_attempt":
                    attempt,

                "attempts":
                    attempts,
            }


        if (
            attempt
            <
            ENVIRONMENT_GATE[
                "retry_count"
            ]
        ):

            time.sleep(
                ENVIRONMENT_GATE[
                    "retry_cooldown_seconds"
                ]
            )


    return {
        "status":
            "INVALID_ENVIRONMENT",

        "successful_attempt":
            None,

        "attempts":
            attempts,
    }


# =============================================================================
# 6. DURABLE SCIENTIFIC PARENT
# =============================================================================

banner(
    "STAGE26-4C3 :: DURABLE SCIENTIFIC PARENT"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-4C3 parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before representation timing."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if RUNTIME_ROOT.exists():

    raise RuntimeError(
        "Stage26-4C3 runtime directory already exists. "
        "Stop for narrow recovery rather than overwriting measurements."
    )


if CHECKPOINT_DIR.exists():

    raise RuntimeError(
        "Stage26-4C3 durable checkpoint already exists."
    )


# =============================================================================
# 7. FROZEN INPUT IDENTITY
# =============================================================================

banner(
    "STAGE26-4C3 :: FROZEN INPUT IDENTITY"
)


identity_checks = [
    (
        "4C1 protocol",
        PROTOCOL_PATH,
        EXPECTED_PROTOCOL_SHA256,
    ),

    (
        "4C1 boundary",
        BOUNDARY_PATH,
        EXPECTED_BOUNDARY_SHA256,
    ),

    (
        "effective worker v2",
        WORKER,
        EXPECTED_WORKER_SHA256,
    ),

    (
        "4C1A erratum",
        ERRATUM_PATH,
        EXPECTED_ERRATUM_SHA256,
    ),

    (
        "4C2 equivalence result",
        EQUIVALENCE_RESULT,
        EXPECTED_EQ_RESULT_SHA256,
    ),

    (
        "4C2 equivalence receipt",
        EQUIVALENCE_RECEIPT,
        EXPECTED_EQ_RECEIPT_SHA256,
    ),

    (
        "4C2 restoration receipt",
        EQUIVALENCE_RESTORATION,
        EXPECTED_EQ_RESTORATION_SHA256,
    ),

    (
        "4C2 equivalence manifest",
        EQUIVALENCE_MANIFEST,
        EXPECTED_EQ_MANIFEST_SHA256,
    ),
]


for label, path, expected in identity_checks:

    if not path.exists():

        raise FileNotFoundError(
            path
        )

    actual = sha256_file(
        path
    )

    passed = (
        actual
        ==
        expected
    )


    print(
        f"{label:32s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Frozen identity mismatch: {label}"
        )


# =============================================================================
# 8. TIMING PERMISSION GATE
# =============================================================================

banner(
    "STAGE26-4C3 :: TIMING PERMISSION GATE"
)


protocol = json.loads(
    PROTOCOL_PATH.read_text(
        encoding="utf-8"
    )
)

erratum = json.loads(
    ERRATUM_PATH.read_text(
        encoding="utf-8"
    )
)

equivalence_result = json.loads(
    EQUIVALENCE_RESULT.read_text(
        encoding="utf-8"
    )
)

equivalence_receipt = json.loads(
    EQUIVALENCE_RECEIPT.read_text(
        encoding="utf-8"
    )
)


permission_checks = {
    "protocol_frozen":
        (
            protocol[
                "status"
            ]
            ==
            "FROZEN_BEFORE_FIRST_REPRESENTATION_MEASUREMENT"
        ),

    "effective_worker_v2":
        (
            erratum[
                "effective_execution_rule"
            ][
                "benchmark_worker"
            ]
            ==
            "stage26_representation_worker_v2.py"
        ),

    "equivalence_PASS":
        (
            equivalence_result[
                "status"
            ]
            ==
            "PASS"
        ),

    "equivalence_fingerprint":
        (
            equivalence_result[
                "representation_fingerprint_sha256"
            ]
            ==
            EXPECTED_EQ_FINGERPRINT
        ),

    "equivalence_untimed":
        (
            equivalence_result[
                "timing_performed"
            ]
            is False
        ),

    "equivalence_receipt_PASS":
        (
            equivalence_receipt[
                "status"
            ]
            ==
            "PASS_EXACT_128_FLOW_REPRESENTATION_EQUIVALENCE"
        ),

    "timing_permission":
        (
            equivalence_receipt[
                "timing_permission_after_git_anchor"
            ]
            is True
        ),

    "labels_forbidden":
        (
            protocol[
                "scientific_rules"
            ][
                "labels_may_be_opened"
            ]
            is False
        ),

    "GPU_false":
        (
            protocol[
                "scientific_rules"
            ][
                "GPU_used"
            ]
            is False
        ),
}


for name, passed in permission_checks.items():

    print(
        f"{name:40s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    permission_checks.values()
):

    raise RuntimeError(
        "Representation timing permission gate failed."
    )


# =============================================================================
# 9. VERIFY FROZEN CONDITION VALUES AGAINST PROTOCOL
# =============================================================================

banner(
    "STAGE26-4C3 :: CONDITION CONSISTENCY"
)


protocol_batches = (
    protocol[
        "sample"
    ][
        "batch_sizes"
    ]
)


if protocol_batches != BATCH_SIZES:

    raise RuntimeError(
        "Batch-size plan differs from frozen protocol."
    )


if int(
    protocol[
        "sample"
    ][
        "start_index_zero_based"
    ]
) != SAMPLE_START:

    raise RuntimeError(
        "Sample start differs from frozen protocol."
    )


if protocol[
    "hardware"
][
    "affinity"
] != CPU_AFFINITY:

    raise RuntimeError(
        "CPU affinity differs from frozen protocol."
    )


if int(
    protocol[
        "hardware"
    ][
        "thread_count"
    ]
) != THREAD_COUNT:

    raise RuntimeError(
        "Thread count differs from frozen protocol."
    )


if int(
    protocol[
        "condition_timeout_seconds"
    ]
) != CONDITION_TIMEOUT_SECONDS:

    raise RuntimeError(
        "Timeout differs from frozen protocol."
    )


for batch in BATCH_SIZES:

    frozen = protocol[
        "iteration_policy"
    ][
        str(
            batch
        )
    ]

    local = ITERATION_POLICY[
        batch
    ]


    if int(
        frozen[
            "warmup_runs"
        ]
    ) != local[
        "warmup_runs"
    ]:

        raise RuntimeError(
            f"Warmup mismatch for B={batch}"
        )


    if int(
        frozen[
            "timed_runs"
        ]
    ) != local[
        "timed_runs"
    ]:

        raise RuntimeError(
            f"Timed-run mismatch for B={batch}"
        )


print(
    "Batch sizes  :",
    BATCH_SIZES
)

print(
    "Sample start :",
    SAMPLE_START
)

print(
    "Affinity     :",
    CPU_AFFINITY
)

print(
    "Threads      :",
    THREAD_COUNT
)

print(
    "Timeout      :",
    CONDITION_TIMEOUT_SECONDS,
    "seconds"
)

print(
    "\nAll frozen condition values: PASS"
)


# =============================================================================
# 10. REPRESENTATION SOURCE PRESENCE / SIZE GATE
# =============================================================================

banner(
    "STAGE26-4C3 :: REPRESENTATION SOURCE GATE"
)


if not CORPUS_RUNTIME_DIR.is_dir():

    raise FileNotFoundError(
        CORPUS_RUNTIME_DIR
    )


runtime_files = sorted(
    path.name
    for path in CORPUS_RUNTIME_DIR.iterdir()
    if path.is_file()
)


expected_runtime_files = sorted(
    SOURCE_IDENTITIES.keys()
)


print(
    "Expected runtime files:",
    expected_runtime_files
)

print(
    "Actual runtime files  :",
    runtime_files
)


if runtime_files != expected_runtime_files:

    raise RuntimeError(
        "Representation runtime source changed after 4C2."
    )


if LABELS_PATH.exists():

    raise RuntimeError(
        "labels.npy unexpectedly exists before timing."
    )


for filename, frozen in SOURCE_IDENTITIES.items():

    path = frozen[
        "path"
    ]

    actual_size = int(
        path.stat().st_size
    )

    passed = (
        actual_size
        ==
        frozen[
            "size_bytes"
        ]
    )


    print(
        f"{filename:22s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:12,d} B"
    )


    if not passed:

        raise RuntimeError(
            f"Runtime source size changed: {filename}"
        )


print(
    "\nFull encoded_bytes.bin rehash before timing: NO"
)

print(
    "Reason: avoid unnecessary page-cache perturbation."
)

print(
    "Exact source hashes are inherited from remotely anchored 4C2 restoration."
)

print(
    "labels.npy present: NO"
)


# =============================================================================
# 11. CREATE RUNTIME ROOT
# =============================================================================

RUNTIME_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

RUNTIME_CONDITIONS.mkdir(
    parents=True,
    exist_ok=False,
)


# =============================================================================
# 12. WRITE PRE-MEASUREMENT EXECUTION PLAN
# =============================================================================

banner(
    "STAGE26-4C3 :: WRITE PRE-MEASUREMENT EXECUTION PLAN"
)


execution_plan = {
    "schema":
        "stage26_4c3_cpu1_representation_execution_plan_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4C3",

    "status":
        "FROZEN_EXECUTION_OF_EXISTING_STAGE26_4C_PROTOCOL",

    "scientific_parent":
        EXPECTED_PARENT,

    "worker_sha256":
        EXPECTED_WORKER_SHA256,

    "protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "boundary_sha256":
        EXPECTED_BOUNDARY_SHA256,

    "equivalence_manifest_sha256":
        EXPECTED_EQ_MANIFEST_SHA256,

    "equivalence_fingerprint_sha256":
        EXPECTED_EQ_FINGERPRINT,

    "condition_order":
        BATCH_SIZES,

    "condition_order_rule":
        "FROZEN_PROTOCOL_BATCH_LIST_ORDER",

    "sample_start_index_zero_based":
        SAMPLE_START,

    "CPU_affinity":
        CPU_AFFINITY,

    "thread_count":
        THREAD_COUNT,

    "fresh_process_per_batch_condition":
        True,

    "batch_conditions": {
        str(
            batch
        ): {
            "batch_size":
                batch,

            "warmup_runs":
                ITERATION_POLICY[
                    batch
                ][
                    "warmup_runs"
                ],

            "timed_runs":
                ITERATION_POLICY[
                    batch
                ][
                    "timed_runs"
                ],

            "start_index_zero_based":
                SAMPLE_START,

            "end_index_exclusive":
                SAMPLE_START
                +
                batch,
        }
        for batch in BATCH_SIZES
    },

    "environment_gate":
        ENVIRONMENT_GATE,

    "condition_timeout_seconds":
        CONDITION_TIMEOUT_SECONDS,

    "timer":
        "time.perf_counter_ns",

    "timer_boundary": {
        "representation_materialize_batch":
            True,

        "process_startup":
            False,

        "source_initialization":
            False,

        "fingerprinting":
            False,

        "JSON_serialization":
            False,

        "float32_div255":
            False,

        "inference":
            False,
    },

    "cache_policy": {
        "exact_batch_warmup_before_timing":
            True,

        "OS_page_cache_flush":
            False,

        "OS_page_cache_manipulation":
            False,

        "full_encoded_file_rehash_before_timing":
            False,

        "cold_storage_claim_allowed":
            False,
    },

    "summary_reporting": {
        "quantile_implementation":
            'numpy.percentile(method="linear")',

        "p50":
            "ALL_CONDITIONS",

        "p95":
            "ALL_CONDITIONS",

        "p99":
            "ONLY_WHEN_TIMED_ITERATION_COUNT_GTE_100",

        "throughput_summary":
            [
                "median",
                "minimum",
                "maximum",
            ],
    },

    "scientific_exclusions": {
        "labels":
            True,

        "pcap":
            True,

        "corpus_regeneration":
            True,

        "raw_masking_encoding":
            True,

        "float32_div255":
            True,

        "models":
            True,

        "inference":
            True,

        "GPU":
            True,
    },

    "measurement_started_when_written":
        False,
}


atomic_json(
    EXECUTION_PLAN_PATH,
    execution_plan,
)


execution_plan_sha = sha256_file(
    EXECUTION_PLAN_PATH
)


print(
    "Execution plan SHA256:"
)

print(
    " ",
    execution_plan_sha
)

print(
    "\nRepresentation measurement has not started yet."
)


# =============================================================================
# 13. WORKER ENVIRONMENT
# =============================================================================

worker_env = os.environ.copy()


for key in [
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
]:

    worker_env[
        key
    ] = str(
        THREAD_COUNT
    )


worker_env[
    "CUDA_VISIBLE_DEVICES"
] = ""


# =============================================================================
# 14. RUN ALL FIVE FROZEN CONDITIONS
# =============================================================================

banner(
    "STAGE26-4C3 :: EXECUTE FROZEN CPU1 REPRESENTATION CONDITIONS"
)


condition_records = []

raw_observation_rows = []


for condition_index, batch in enumerate(
    BATCH_SIZES,
    start=1,
):

    print(
        "\n"
        +
        "-" * 122
    )

    print(
        f"CONDITION {condition_index}/5 :: "
        f"CPU1 :: B={batch}"
    )

    print(
        "-" * 122
    )


    policy = ITERATION_POLICY[
        batch
    ]


    # -------------------------------------------------------------------------
    # Environment gate
    # -------------------------------------------------------------------------

    gate = environment_gate()


    if gate[
        "status"
    ] != "PASS":

        raise RuntimeError(
            f"Frozen environment gate failed for B={batch}. "
            "Do not rerun completed conditions; use narrow recovery."
        )


    successful_gate = gate[
        "attempts"
    ][
        gate[
            "successful_attempt"
        ]
        -
        1
    ]


    # -------------------------------------------------------------------------
    # Condition paths
    # -------------------------------------------------------------------------

    condition_dir = (
        RUNTIME_CONDITIONS
        /
        f"batch_{batch}"
    )

    condition_dir.mkdir(
        parents=True,
        exist_ok=False,
    )


    config_path = (
        condition_dir
        /
        f"stage26_4c3_B{batch}_config.json"
    )

    result_path = (
        condition_dir
        /
        f"stage26_4c3_B{batch}_result.json"
    )


    # -------------------------------------------------------------------------
    # Frozen worker config
    # -------------------------------------------------------------------------

    config = {
        "schema":
            "stage26_4c3_cpu1_representation_condition_config_v1",

        "mode":
            "BENCHMARK_TIMED",

        "scientific_parent":
            EXPECTED_PARENT,

        "worker_sha256":
            EXPECTED_WORKER_SHA256,

        "protocol_sha256":
            EXPECTED_PROTOCOL_SHA256,

        "equivalence_fingerprint_sha256":
            EXPECTED_EQ_FINGERPRINT,

        "corpus_dir":
            str(
                CORPUS_RUNTIME_DIR
            ),

        "expected_flow_count":
            FLOW_COUNT,

        "start_index":
            SAMPLE_START,

        "batch_size":
            batch,

        "end_index_exclusive":
            SAMPLE_START
            +
            batch,

        "warmup_runs":
            policy[
                "warmup_runs"
            ],

        "timed_runs":
            policy[
                "timed_runs"
            ],

        "affinity":
            CPU_AFFINITY,

        "thread_count":
            THREAD_COUNT,

        "result_path":
            str(
                result_path
            ),

        "labels_allowed":
            False,

        "float32_div255_allowed":
            False,

        "models_allowed":
            False,

        "gpu_allowed":
            False,
    }


    atomic_json(
        config_path,
        config,
    )


    config_sha = sha256_file(
        config_path
    )


    print(
        f"  warmup runs : {policy['warmup_runs']}"
    )

    print(
        f"  timed runs  : {policy['timed_runs']}"
    )

    print(
        f"  flow range  : "
        f"[{SAMPLE_START}, {SAMPLE_START + batch})"
    )

    print(
        f"  config SHA  : {config_sha}"
    )


    if result_path.exists():

        raise RuntimeError(
            f"Result already exists for B={batch}."
        )


    # -------------------------------------------------------------------------
    # Fresh-process execution
    # -------------------------------------------------------------------------

    process_started_utc = datetime.now(
        timezone.utc
    ).isoformat()


    try:

        worker_run = run(
            [
                sys.executable,
                str(
                    WORKER
                ),
                str(
                    config_path
                ),
            ],
            cwd=REPO,
            env=worker_env,
            check=False,
            text=True,
            timeout=CONDITION_TIMEOUT_SECONDS,
        )


    except subprocess.TimeoutExpired as exc:

        raise RuntimeError(
            f"TIMEOUT_RESOURCE_LIMIT for B={batch} after "
            f"{CONDITION_TIMEOUT_SECONDS}s. "
            "Do not adapt the frozen condition."
        ) from exc


    process_finished_utc = datetime.now(
        timezone.utc
    ).isoformat()


    print(
        "  worker return code:",
        worker_run.returncode
    )


    if worker_run.stdout.strip():

        print(
            "\n  worker stdout/stderr:"
        )

        print(
            worker_run.stdout
        )


    if worker_run.returncode != 0:

        if worker_run.returncode in {
            -9,
            137,
        }:

            raise RuntimeError(
                f"RESOURCE_LIMIT_OOM-like worker termination "
                f"for B={batch}, return code {worker_run.returncode}. "
                "Do not adapt batch size."
            )


        raise RuntimeError(
            f"IMPLEMENTATION_FAILURE for B={batch}, "
            f"return code {worker_run.returncode}. "
            "Perform narrow diagnosis only."
        )


    if not result_path.exists():

        raise RuntimeError(
            f"Worker B={batch} produced no result file."
        )


    result_sha = sha256_file(
        result_path
    )

    result = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


    # -------------------------------------------------------------------------
    # Strict scientific result audit
    # -------------------------------------------------------------------------

    expected_image_bytes = (
        batch
        *
        ROWS
        *
        COLS
    )

    expected_mask_bytes = (
        batch
        *
        ROWS
        *
        COLS
    )

    expected_total_bytes = (
        expected_image_bytes
        +
        expected_mask_bytes
    )


    result_checks = {
        "status_PASS":
            (
                result[
                    "status"
                ]
                ==
                "PASS"
            ),

        "mode":
            (
                result[
                    "mode"
                ]
                ==
                "BENCHMARK_TIMED"
            ),

        "batch":
            (
                int(
                    result[
                        "batch_size"
                    ]
                )
                ==
                batch
            ),

        "start":
            (
                int(
                    result[
                        "start_index"
                    ]
                )
                ==
                SAMPLE_START
            ),

        "end":
            (
                int(
                    result[
                        "end_index_exclusive"
                    ]
                )
                ==
                SAMPLE_START
                +
                batch
            ),

        "population":
            (
                int(
                    result[
                        "population_flow_count"
                    ]
                )
                ==
                FLOW_COUNT
            ),

        "warmups":
            (
                int(
                    result[
                        "warmup_runs"
                    ]
                )
                ==
                policy[
                    "warmup_runs"
                ]
            ),

        "timed_runs":
            (
                int(
                    result[
                        "timed_runs"
                    ]
                )
                ==
                policy[
                    "timed_runs"
                ]
            ),

        "observation_count":
            (
                len(
                    result[
                        "observations"
                    ]
                )
                ==
                policy[
                    "timed_runs"
                ]
            ),

        "image_shape":
            (
                result[
                    "image_shape_per_flow"
                ]
                ==
                [
                    64,
                    256,
                ]
            ),

        "image_dtype":
            (
                result[
                    "image_dtype"
                ]
                ==
                "uint8"
            ),

        "mask_shape":
            (
                result[
                    "padding_mask_shape_per_flow"
                ]
                ==
                [
                    64,
                    256,
                ]
            ),

        "mask_dtype":
            (
                result[
                    "padding_mask_dtype"
                ]
                ==
                "bool"
            ),

        "timing_true":
            (
                result[
                    "timing_performed"
                ]
                is True
            ),

        "div255_false":
            (
                result[
                    "float32_div255_performed"
                ]
                is False
            ),

        "model_false":
            (
                result[
                    "model_loaded"
                ]
                is False
            ),

        "labels_false":
            (
                result[
                    "labels_accessed"
                ]
                is False
            ),

        "pcap_false":
            (
                result[
                    "pcap_accessed"
                ]
                is False
            ),

        "corpus_created_false":
            (
                result[
                    "corpus_created_or_modified"
                ]
                is False
            ),

        "GPU_false":
            (
                result[
                    "gpu_used"
                ]
                is False
            ),
    }


    for name, passed in result_checks.items():

        if not passed:

            raise RuntimeError(
                f"B={batch} result audit failed: {name}"
            )


    fingerprint = result[
        "representation_fingerprint_sha256"
    ]

    warm_fingerprint = result[
        "warm_representation_fingerprint_sha256"
    ]


    if (
        not isinstance(
            fingerprint,
            str,
        )
        or
        len(
            fingerprint
        )
        != 64
    ):

        raise RuntimeError(
            f"Invalid representation fingerprint B={batch}."
        )


    if warm_fingerprint != fingerprint:

        raise RuntimeError(
            f"Warm/timed representation fingerprint mismatch B={batch}."
        )


    # -------------------------------------------------------------------------
    # Audit every raw observation
    # -------------------------------------------------------------------------

    elapsed_seconds_values = []

    flows_per_second_values = []


    for expected_iteration, observation in enumerate(
        result[
            "observations"
        ],
        start=1,
    ):

        if int(
            observation[
                "iteration_index"
            ]
        ) != expected_iteration:

            raise RuntimeError(
                f"B={batch}: observation iteration ordering mismatch."
            )


        elapsed_ns = int(
            observation[
                "elapsed_ns"
            ]
        )

        elapsed_seconds = float(
            observation[
                "elapsed_seconds"
            ]
        )

        flows = int(
            observation[
                "flows"
            ]
        )

        flows_per_second = float(
            observation[
                "flows_per_second"
            ]
        )


        if elapsed_ns <= 0:

            raise RuntimeError(
                f"B={batch}: non-positive elapsed_ns."
            )


        if elapsed_seconds <= 0:

            raise RuntimeError(
                f"B={batch}: non-positive elapsed_seconds."
            )


        if flows != batch:

            raise RuntimeError(
                f"B={batch}: flow-count mismatch."
            )


        expected_throughput = (
            batch
            /
            elapsed_seconds
        )


        if not math.isclose(
            flows_per_second,
            expected_throughput,
            rel_tol=1e-12,
            abs_tol=1e-12,
        ):

            raise RuntimeError(
                f"B={batch}: throughput arithmetic mismatch."
            )


        if int(
            observation[
                "image_output_bytes"
            ]
        ) != expected_image_bytes:

            raise RuntimeError(
                f"B={batch}: image byte accounting mismatch."
            )


        if int(
            observation[
                "padding_mask_output_bytes"
            ]
        ) != expected_mask_bytes:

            raise RuntimeError(
                f"B={batch}: mask byte accounting mismatch."
            )


        if int(
            observation[
                "total_output_bytes"
            ]
        ) != expected_total_bytes:

            raise RuntimeError(
                f"B={batch}: total byte accounting mismatch."
            )


        elapsed_seconds_values.append(
            elapsed_seconds
        )

        flows_per_second_values.append(
            flows_per_second
        )


        raw_observation_rows.append(
            {
                "condition_index":
                    condition_index,

                "batch_size":
                    batch,

                "iteration_index":
                    expected_iteration,

                "elapsed_ns":
                    elapsed_ns,

                "elapsed_seconds":
                    elapsed_seconds,

                "flows":
                    flows,

                "flows_per_second":
                    flows_per_second,

                "image_output_bytes":
                    expected_image_bytes,

                "padding_mask_output_bytes":
                    expected_mask_bytes,

                "total_output_bytes":
                    expected_total_bytes,
            }
        )


    # -------------------------------------------------------------------------
    # Frozen summary metrics
    # -------------------------------------------------------------------------

    timed_count = len(
        elapsed_seconds_values
    )


    p50_seconds = percentile(
        elapsed_seconds_values,
        50,
    )

    p95_seconds = percentile(
        elapsed_seconds_values,
        95,
    )


    if timed_count >= 100:

        p99_seconds = percentile(
            elapsed_seconds_values,
            99,
        )

    else:

        p99_seconds = None


    median_fps = float(
        np.median(
            np.asarray(
                flows_per_second_values,
                dtype=np.float64,
            )
        )
    )

    min_fps = float(
        np.min(
            np.asarray(
                flows_per_second_values,
                dtype=np.float64,
            )
        )
    )

    max_fps = float(
        np.max(
            np.asarray(
                flows_per_second_values,
                dtype=np.float64,
            )
        )
    )


    condition_record = {
        "condition_index":
            condition_index,

        "status":
            "PASS",

        "batch_size":
            batch,

        "sample_start_index_zero_based":
            SAMPLE_START,

        "sample_end_index_exclusive":
            SAMPLE_START
            +
            batch,

        "warmup_runs":
            policy[
                "warmup_runs"
            ],

        "timed_runs":
            timed_count,

        "CPU_affinity":
            CPU_AFFINITY,

        "thread_count":
            THREAD_COUNT,

        "environment_gate":
            gate,

        "successful_gate_cpu_percent":
            successful_gate[
                "cpu_utilization_percent"
            ],

        "successful_gate_available_ram_gib":
            successful_gate[
                "available_ram_gib"
            ],

        "process_started_utc":
            process_started_utc,

        "process_finished_utc":
            process_finished_utc,

        "config_path":
            str(
                config_path
            ),

        "config_sha256":
            config_sha,

        "result_path":
            str(
                result_path
            ),

        "result_sha256":
            result_sha,

        "representation_fingerprint_sha256":
            fingerprint,

        "output_bytes_per_batch": {
            "image":
                expected_image_bytes,

            "padding_mask":
                expected_mask_bytes,

            "total":
                expected_total_bytes,
        },

        "summary": {
            "p50_batch_latency_seconds":
                p50_seconds,

            "p95_batch_latency_seconds":
                p95_seconds,

            "p99_batch_latency_seconds":
                p99_seconds,

            "median_flows_per_second":
                median_fps,

            "minimum_flows_per_second":
                min_fps,

            "maximum_flows_per_second":
                max_fps,
        },
    }


    condition_records.append(
        condition_record
    )


    print(
        f"\n  RESULT B={batch}: PASS"
    )

    print(
        f"  result SHA       : {result_sha}"
    )

    print(
        f"  fingerprint      : {fingerprint}"
    )

    print(
        f"  p50 latency      : "
        f"{p50_seconds * 1000:.6f} ms"
    )

    print(
        f"  p95 latency      : "
        f"{p95_seconds * 1000:.6f} ms"
    )


    if p99_seconds is not None:

        print(
            f"  p99 latency      : "
            f"{p99_seconds * 1000:.6f} ms"
        )

    else:

        print(
            "  p99 latency      : NOT REPORTED (n < 100)"
        )


    print(
        f"  median flows/s   : {median_fps:,.3f}"
    )

    print(
        f"  min flows/s      : {min_fps:,.3f}"
    )

    print(
        f"  max flows/s      : {max_fps:,.3f}"
    )


# =============================================================================
# 15. POST-MEASUREMENT SOURCE / LABEL SAFETY GATE
# =============================================================================

banner(
    "STAGE26-4C3 :: POST-MEASUREMENT SAFETY GATE"
)


if LABELS_PATH.exists():

    raise RuntimeError(
        "labels.npy appeared during representation timing."
    )


post_runtime_files = sorted(
    path.name
    for path in CORPUS_RUNTIME_DIR.iterdir()
    if path.is_file()
)


if post_runtime_files != expected_runtime_files:

    raise RuntimeError(
        "Representation source file set changed during timing."
    )


for filename, frozen in SOURCE_IDENTITIES.items():

    if int(
        frozen[
            "path"
        ].stat().st_size
    ) != frozen[
        "size_bytes"
    ]:

        raise RuntimeError(
            f"Runtime representation source size changed: {filename}"
        )


print(
    "labels.npy present        : NO"
)

print(
    "runtime source file set   : UNCHANGED"
)

print(
    "runtime source file sizes : UNCHANGED"
)

print(
    "PCAP access               : NO"
)

print(
    "model loading             : NO"
)

print(
    "GPU                       : NO"
)


# =============================================================================
# 16. RAW OBSERVATION CSV
# =============================================================================

banner(
    "STAGE26-4C3 :: WRITE RAW OBSERVATIONS"
)


raw_fieldnames = [
    "condition_index",
    "batch_size",
    "iteration_index",
    "elapsed_ns",
    "elapsed_seconds",
    "flows",
    "flows_per_second",
    "image_output_bytes",
    "padding_mask_output_bytes",
    "total_output_bytes",
]


with RAW_CSV_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=raw_fieldnames,
    )

    writer.writeheader()

    writer.writerows(
        raw_observation_rows
    )


raw_csv_sha = sha256_file(
    RAW_CSV_PATH
)


expected_raw_count = sum(
    ITERATION_POLICY[
        batch
    ][
        "timed_runs"
    ]
    for batch in BATCH_SIZES
)


if len(
    raw_observation_rows
) != expected_raw_count:

    raise RuntimeError(
        "Unexpected total raw observation count."
    )


print(
    "Raw observations:",
    len(
        raw_observation_rows
    )
)

print(
    "Expected        :",
    expected_raw_count
)

print(
    "Raw CSV SHA256 :",
    raw_csv_sha
)


# =============================================================================
# 17. SUMMARY JSON + CSV
# =============================================================================

banner(
    "STAGE26-4C3 :: WRITE FROZEN SUMMARIES"
)


summary = {
    "schema":
        "stage26_4c3_cpu1_representation_summary_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4C3",

    "status":
        "PASS_ALL_FIVE_FROZEN_CPU1_REPRESENTATION_CONDITIONS",

    "scientific_parent":
        EXPECTED_PARENT,

    "worker_sha256":
        EXPECTED_WORKER_SHA256,

    "protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "equivalence_fingerprint_sha256":
        EXPECTED_EQ_FINGERPRINT,

    "execution_plan_sha256":
        execution_plan_sha,

    "CPU_mode":
        "CPU_1_PHYSICAL_CORE",

    "CPU_affinity":
        CPU_AFFINITY,

    "thread_count":
        THREAD_COUNT,

    "sample_start_index_zero_based":
        SAMPLE_START,

    "condition_count":
        len(
            condition_records
        ),

    "raw_observation_count":
        len(
            raw_observation_rows
        ),

    "quantile_method":
        "numpy.percentile(method=linear)",

    "conditions":
        condition_records,

    "scientific_boundary": {
        "component":
            (
                "existing Stage20 compact Release corpus "
                "-> dense uint8 image + bool byte-padding mask"
            ),

        "warm_representation_materialization":
            True,

        "labels_accessed":
            False,

        "float32_div255_included":
            False,

        "model_inference_included":
            False,

        "release_tar_restoration_included":
            False,

        "process_startup_included":
            False,

        "source_initialization_included":
            False,

        "fingerprinting_included":
            False,

        "page_cache_manipulated":
            False,

        "cold_storage_claim_allowed":
            False,

        "raw_packet_masking_encoding_included":
            False,

        "complete_E2E_claim_allowed":
            False,
    },
}


atomic_json(
    SUMMARY_JSON_PATH,
    summary,
)


summary_json_sha = sha256_file(
    SUMMARY_JSON_PATH
)


summary_csv_fields = [
    "batch_size",
    "warmup_runs",
    "timed_runs",
    "p50_batch_latency_seconds",
    "p95_batch_latency_seconds",
    "p99_batch_latency_seconds",
    "median_flows_per_second",
    "minimum_flows_per_second",
    "maximum_flows_per_second",
    "successful_gate_cpu_percent",
    "successful_gate_available_ram_gib",
    "representation_fingerprint_sha256",
    "result_sha256",
]


with SUMMARY_CSV_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=summary_csv_fields,
    )

    writer.writeheader()


    for record in condition_records:

        s = record[
            "summary"
        ]


        writer.writerow(
            {
                "batch_size":
                    record[
                        "batch_size"
                    ],

                "warmup_runs":
                    record[
                        "warmup_runs"
                    ],

                "timed_runs":
                    record[
                        "timed_runs"
                    ],

                "p50_batch_latency_seconds":
                    s[
                        "p50_batch_latency_seconds"
                    ],

                "p95_batch_latency_seconds":
                    s[
                        "p95_batch_latency_seconds"
                    ],

                "p99_batch_latency_seconds":
                    (
                        ""
                        if s[
                            "p99_batch_latency_seconds"
                        ]
                        is None
                        else
                        s[
                            "p99_batch_latency_seconds"
                        ]
                    ),

                "median_flows_per_second":
                    s[
                        "median_flows_per_second"
                    ],

                "minimum_flows_per_second":
                    s[
                        "minimum_flows_per_second"
                    ],

                "maximum_flows_per_second":
                    s[
                        "maximum_flows_per_second"
                    ],

                "successful_gate_cpu_percent":
                    record[
                        "successful_gate_cpu_percent"
                    ],

                "successful_gate_available_ram_gib":
                    record[
                        "successful_gate_available_ram_gib"
                    ],

                "representation_fingerprint_sha256":
                    record[
                        "representation_fingerprint_sha256"
                    ],

                "result_sha256":
                    record[
                        "result_sha256"
                    ],
            }
        )


summary_csv_sha = sha256_file(
    SUMMARY_CSV_PATH
)


print(
    "Summary JSON SHA256:",
    summary_json_sha
)

print(
    "Summary CSV SHA256 :",
    summary_csv_sha
)


# =============================================================================
# 18. SCIENTIFIC RECEIPT
# =============================================================================

banner(
    "STAGE26-4C3 :: WRITE SCIENTIFIC RECEIPT"
)


receipt = {
    "schema":
        "stage26_4c3_cpu1_representation_timing_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-4C3",

    "status":
        "PASS_FIVE_FROZEN_CPU1_REPRESENTATION_CONDITIONS",

    "scientific_parent":
        EXPECTED_PARENT,

    "frozen_inputs": {
        "protocol_sha256":
            EXPECTED_PROTOCOL_SHA256,

        "boundary_sha256":
            EXPECTED_BOUNDARY_SHA256,

        "worker_v2_sha256":
            EXPECTED_WORKER_SHA256,

        "erratum_sha256":
            EXPECTED_ERRATUM_SHA256,

        "equivalence_result_sha256":
            EXPECTED_EQ_RESULT_SHA256,

        "equivalence_receipt_sha256":
            EXPECTED_EQ_RECEIPT_SHA256,

        "equivalence_manifest_sha256":
            EXPECTED_EQ_MANIFEST_SHA256,

        "equivalence_fingerprint_sha256":
            EXPECTED_EQ_FINGERPRINT,

        "execution_plan_sha256":
            execution_plan_sha,
    },

    "source_provenance": {
        "runtime_directory":
            str(
                CORPUS_RUNTIME_DIR
            ),

        "encoded_bytes": {
            "size_bytes":
                SOURCE_IDENTITIES[
                    "encoded_bytes.bin"
                ][
                    "size_bytes"
                ],

            "anchored_sha256":
                SOURCE_IDENTITIES[
                    "encoded_bytes.bin"
                ][
                    "sha256"
                ],
        },

        "flow_offsets": {
            "size_bytes":
                SOURCE_IDENTITIES[
                    "flow_offsets.npy"
                ][
                    "size_bytes"
                ],

            "anchored_sha256":
                SOURCE_IDENTITIES[
                    "flow_offsets.npy"
                ][
                    "sha256"
                ],
        },

        "packet_lengths": {
            "size_bytes":
                SOURCE_IDENTITIES[
                    "packet_lengths.npy"
                ][
                    "size_bytes"
                ],

            "anchored_sha256":
                SOURCE_IDENTITIES[
                    "packet_lengths.npy"
                ][
                    "sha256"
                ],
        },

        "full_encoded_file_rehashed_immediately_before_timing":
            False,

        "reason_full_rehash_not_repeated":
            (
                "Exact member hashes were already verified and remotely "
                "anchored in Stage26-4C2. Re-reading the full 522,845,159-byte "
                "encoded member immediately before warm timing would "
                "unnecessarily perturb OS page-cache state."
            ),
    },

    "measurement": {
        "CPU_mode":
            "CPU_1_PHYSICAL_CORE",

        "CPU_affinity":
            CPU_AFFINITY,

        "thread_count":
            THREAD_COUNT,

        "sample_start_index_zero_based":
            SAMPLE_START,

        "batch_sizes":
            BATCH_SIZES,

        "fresh_process_per_batch_condition":
            True,

        "condition_count":
            5,

        "raw_observation_count":
            len(
                raw_observation_rows
            ),

        "all_conditions_passed":
            True,

        "quantile_method":
            "numpy.percentile(method=linear)",

        "conditions":
            condition_records,
    },

    "output_hashes": {
        "raw_observations_csv_sha256":
            raw_csv_sha,

        "summary_json_sha256":
            summary_json_sha,

        "summary_csv_sha256":
            summary_csv_sha,
    },

    "scientific_boundaries": {
        "representation_performance_measured":
            True,

        "measurement_type":
            "WARM_COMPACT_CORPUS_DENSE_MATERIALIZATION_CPU1",

        "disk_memmap_reads_in_timer":
            True,

        "dense_batch_allocation_in_timer":
            True,

        "release_tar_extraction_in_timer":
            False,

        "Python_startup_in_timer":
            False,

        "source_initialization_in_timer":
            False,

        "fingerprinting_in_timer":
            False,

        "float32_div255_in_timer":
            False,

        "model_inference_in_timer":
            False,

        "labels_accessed":
            False,

        "pcap_accessed":
            False,

        "release_corpus_regenerated":
            False,

        "raw_packet_masking_encoding_measured":
            False,

        "group_A_70_feature_extraction_measured":
            False,

        "complete_E2E_claim_allowed":
            False,

        "page_cache_manipulated":
            False,

        "cold_storage_claim_allowed":
            False,

        "GPU_used":
            False,
    },

    "claim":
        (
            "CPU1 warm materialization throughput/latency for converting the "
            "existing authoritative Stage20 compact packet-image corpus into "
            "dense uint8 [B,64,256] images and bool [B,64,256] byte-padding "
            "masks on the frozen deterministic Monday sample."
        ),

    "explicit_non_claims": [
        "original raw packet masking/encoding throughput",
        "raw PCAP-to-packet-image end-to-end latency",
        "Group-A CICFlowMeter 70-feature extraction throughput",
        "cold-storage corpus-read performance",
        "model-input float32 /255 conversion cost",
        "CNN inference cost",
        "ViT inference cost",
        "complete pipeline end-to-end latency",
    ],
}


atomic_json(
    RECEIPT_RUNTIME_PATH,
    receipt,
)


receipt_sha = sha256_file(
    RECEIPT_RUNTIME_PATH
)


print(
    "Timing receipt SHA256:"
)

print(
    " ",
    receipt_sha
)


# =============================================================================
# 19. PRINT SCIENTIFIC SUMMARY BEFORE GIT
# =============================================================================

banner(
    "STAGE26-4C3 :: CPU1 REPRESENTATION SUMMARY"
)


print(
    f"{'B':>6s} "
    f"{'n':>5s} "
    f"{'p50 ms':>14s} "
    f"{'p95 ms':>14s} "
    f"{'p99 ms':>14s} "
    f"{'median flows/s':>18s}"
)


for record in condition_records:

    s = record[
        "summary"
    ]


    p99_text = (
        "N/A"
        if s[
            "p99_batch_latency_seconds"
        ]
        is None
        else
        f"{s['p99_batch_latency_seconds'] * 1000:.6f}"
    )


    print(
        f"{record['batch_size']:6d} "
        f"{record['timed_runs']:5d} "
        f"{s['p50_batch_latency_seconds'] * 1000:14.6f} "
        f"{s['p95_batch_latency_seconds'] * 1000:14.6f} "
        f"{p99_text:>14s} "
        f"{s['median_flows_per_second']:18,.3f}"
    )


# =============================================================================
# 20. CREATE DURABLE CHECKPOINT
# =============================================================================

banner(
    "STAGE26-4C3 :: CREATE DURABLE CHECKPOINT"
)


CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

CHECKPOINT_CONDITIONS.mkdir(
    parents=True,
    exist_ok=False,
)


# -----------------------------------------------------------------------------
# Copy top-level artifacts
# -----------------------------------------------------------------------------

for source, destination in [
    (
        EXECUTION_PLAN_PATH,
        CHECKPOINT_EXECUTION_PLAN,
    ),

    (
        RAW_CSV_PATH,
        CHECKPOINT_RAW_CSV,
    ),

    (
        SUMMARY_JSON_PATH,
        CHECKPOINT_SUMMARY_JSON,
    ),

    (
        SUMMARY_CSV_PATH,
        CHECKPOINT_SUMMARY_CSV,
    ),

    (
        RECEIPT_RUNTIME_PATH,
        CHECKPOINT_RECEIPT,
    ),
]:

    shutil.copy2(
        source,
        destination,
    )


# -----------------------------------------------------------------------------
# Copy each exact condition config/result
# -----------------------------------------------------------------------------

condition_checkpoint_files = []


for batch in BATCH_SIZES:

    runtime_condition_dir = (
        RUNTIME_CONDITIONS
        /
        f"batch_{batch}"
    )

    checkpoint_condition_dir = (
        CHECKPOINT_CONDITIONS
        /
        f"batch_{batch}"
    )

    checkpoint_condition_dir.mkdir(
        parents=True,
        exist_ok=False,
    )


    for filename in [
        f"stage26_4c3_B{batch}_config.json",
        f"stage26_4c3_B{batch}_result.json",
    ]:

        source = (
            runtime_condition_dir
            /
            filename
        )

        destination = (
            checkpoint_condition_dir
            /
            filename
        )


        shutil.copy2(
            source,
            destination,
        )


        if sha256_file(
            source
        ) != sha256_file(
            destination
        ):

            raise RuntimeError(
                f"Checkpoint copy mismatch: {filename}"
            )


        condition_checkpoint_files.append(
            destination
        )


# Verify top-level byte copies.
for source, destination in [
    (
        EXECUTION_PLAN_PATH,
        CHECKPOINT_EXECUTION_PLAN,
    ),

    (
        RAW_CSV_PATH,
        CHECKPOINT_RAW_CSV,
    ),

    (
        SUMMARY_JSON_PATH,
        CHECKPOINT_SUMMARY_JSON,
    ),

    (
        SUMMARY_CSV_PATH,
        CHECKPOINT_SUMMARY_CSV,
    ),

    (
        RECEIPT_RUNTIME_PATH,
        CHECKPOINT_RECEIPT,
    ),
]:

    if sha256_file(
        source
    ) != sha256_file(
        destination
    ):

        raise RuntimeError(
            f"Checkpoint top-level copy mismatch: {source.name}"
        )


# =============================================================================
# 21. BUILD DURABLE MANIFEST
# =============================================================================

banner(
    "STAGE26-4C3 :: BUILD DURABLE MANIFEST"
)


checkpoint_files = [
    CHECKPOINT_EXECUTION_PLAN,
    CHECKPOINT_RAW_CSV,
    CHECKPOINT_SUMMARY_JSON,
    CHECKPOINT_SUMMARY_CSV,
    CHECKPOINT_RECEIPT,
    *condition_checkpoint_files,
]


manifest_rows = []


for path in checkpoint_files:

    manifest_rows.append(
        {
            "repo_relative_path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest = {
    "schema":
        "stage26_4c3_cpu1_representation_timing_manifest_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        "PASS_FIVE_FROZEN_CONDITIONS_READY_FOR_GIT_ANCHOR",

    "parent_commit":
        EXPECTED_PARENT,

    "commit_subject":
        COMMIT_SUBJECT,

    "protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "boundary_sha256":
        EXPECTED_BOUNDARY_SHA256,

    "worker_sha256":
        EXPECTED_WORKER_SHA256,

    "erratum_sha256":
        EXPECTED_ERRATUM_SHA256,

    "equivalence_manifest_sha256":
        EXPECTED_EQ_MANIFEST_SHA256,

    "equivalence_fingerprint_sha256":
        EXPECTED_EQ_FINGERPRINT,

    "execution_plan_sha256":
        execution_plan_sha,

    "raw_observations_csv_sha256":
        raw_csv_sha,

    "summary_json_sha256":
        summary_json_sha,

    "summary_csv_sha256":
        summary_csv_sha,

    "timing_receipt_sha256":
        receipt_sha,

    "condition_count":
        5,

    "raw_observation_count":
        expected_raw_count,

    "file_count_excluding_manifest":
        len(
            manifest_rows
        ),

    "files":
        manifest_rows,

    "scientific_boundaries": {
        "CPU1_representation_timing_complete":
            True,

        "all_five_conditions_passed":
            True,

        "labels_accessed":
            False,

        "pcap_accessed":
            False,

        "release_corpus_regenerated":
            False,

        "float32_div255_included":
            False,

        "model_inference_included":
            False,

        "GPU_used":
            False,

        "complete_E2E_claim_allowed":
            False,
    },
}


atomic_json(
    CHECKPOINT_MANIFEST,
    manifest,
)


manifest_sha = sha256_file(
    CHECKPOINT_MANIFEST
)


print(
    "Execution plan :",
    execution_plan_sha
)

print(
    "Raw CSV        :",
    raw_csv_sha
)

print(
    "Summary JSON   :",
    summary_json_sha
)

print(
    "Summary CSV    :",
    summary_csv_sha
)

print(
    "Receipt        :",
    receipt_sha
)

print(
    "Manifest       :",
    manifest_sha
)

print(
    "Files          :",
    len(
        manifest_rows
    ),
    "+ manifest"
)


# =============================================================================
# 22. LOCAL DURABLE PACKAGE AUDIT
# =============================================================================

banner(
    "STAGE26-4C3 :: LOCAL DURABLE PACKAGE AUDIT"
)


for row in manifest_rows:

    path = (
        REPO
        /
        row[
            "repo_relative_path"
        ]
    )

    actual_size = int(
        path.stat().st_size
    )

    actual_sha = sha256_file(
        path
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Local Stage26-4C3 package audit failed."
        )


# =============================================================================
# 23. GIT CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-4C3 :: GIT CHANGE AUDIT"
)


repo_status = git(
    "status",
    "--porcelain",
)


print(
    repo_status
)


if not repo_status:

    raise RuntimeError(
        "Expected uncommitted Stage26-4C3 checkpoint."
    )


unexpected = []


for line in repo_status.splitlines():

    rel = line[
        3:
    ]


    if not rel.startswith(
        str(
            CHECKPOINT_REL
        )
        +
        "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository changes:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 24. GIT IDENTITY
# =============================================================================

author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_PARENT,
)

author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_PARENT,
)


if (
    not author_name.strip()
    or
    "@"
    not in
    author_email
):

    raise RuntimeError(
        "Could not resolve Git author identity."
    )


git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


print(
    "\nGit author:"
)

print(
    " ",
    author_name,
    "<" + author_email + ">"
)


# =============================================================================
# 25. COMMIT
# =============================================================================

banner(
    "STAGE26-4C3 :: COMMIT"
)


git(
    "add",
    str(
        CHECKPOINT_REL
    ),
)


print(
    git(
        "diff",
        "--cached",
        "--name-status",
    )
)


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-4C3 commit parent mismatch."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Stage26-4C3 commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository not clean after Stage26-4C3 commit."
    )


# =============================================================================
# 26. PUSH
# =============================================================================

banner(
    "STAGE26-4C3 :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    pushed = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
        check=True,
        text=True,
    )


    print(
        pushed.stdout.strip()
    )


github_token = None


# =============================================================================
# 27. REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-4C3 :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "origin/main != Stage26-4C3 commit."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote Stage26-4C3 parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote Stage26-4C3 subject mismatch."
    )


# =============================================================================
# 28. REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-4C3 :: REMOTE BYTE VERIFICATION"
)


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    str(
        CHECKPOINT_MANIFEST.relative_to(
            REPO
        )
    ),
)

remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "Remote manifest SHA256:"
)

print(
    " ",
    remote_manifest_sha
)


if remote_manifest_sha != manifest_sha:

    raise RuntimeError(
        "Remote Stage26-4C3 manifest mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


if (
    int(
        remote_manifest[
            "file_count_excluding_manifest"
        ]
    )
    !=
    len(
        manifest_rows
    )
):

    raise RuntimeError(
        "Unexpected remote Stage26-4C3 file count."
    )


for row in remote_manifest[
    "files"
]:

    data = git_blob_bytes(
        "origin/main",
        row[
            "repo_relative_path"
        ],
    )

    actual_size = len(
        data
    )

    actual_sha = sha256_bytes(
        data
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Remote Stage26-4C3 byte verification failed."
        )


# =============================================================================
# 29. REMOTE SCIENTIFIC AUDIT
# =============================================================================

banner(
    "STAGE26-4C3 :: REMOTE SCIENTIFIC AUDIT"
)


remote_summary = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            CHECKPOINT_SUMMARY_JSON.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)


remote_receipt = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            CHECKPOINT_RECEIPT.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)


remote_checks = {
    "summary_PASS":
        (
            remote_summary[
                "status"
            ]
            ==
            "PASS_ALL_FIVE_FROZEN_CPU1_REPRESENTATION_CONDITIONS"
        ),

    "five_conditions":
        (
            int(
                remote_summary[
                    "condition_count"
                ]
            )
            ==
            5
        ),

    "520_raw_observations":
        (
            int(
                remote_summary[
                    "raw_observation_count"
                ]
            )
            ==
            expected_raw_count
        ),

    "CPU1":
        (
            remote_summary[
                "CPU_mode"
            ]
            ==
            "CPU_1_PHYSICAL_CORE"
        ),

    "affinity_0":
        (
            remote_summary[
                "CPU_affinity"
            ]
            ==
            [
                0
            ]
        ),

    "receipt_PASS":
        (
            remote_receipt[
                "status"
            ]
            ==
            "PASS_FIVE_FROZEN_CPU1_REPRESENTATION_CONDITIONS"
        ),

    "representation_measured":
        (
            remote_receipt[
                "scientific_boundaries"
            ][
                "representation_performance_measured"
            ]
            is True
        ),

    "labels_false":
        (
            remote_receipt[
                "scientific_boundaries"
            ][
                "labels_accessed"
            ]
            is False
        ),

    "pcap_false":
        (
            remote_receipt[
                "scientific_boundaries"
            ][
                "pcap_accessed"
            ]
            is False
        ),

    "corpus_regeneration_false":
        (
            remote_receipt[
                "scientific_boundaries"
            ][
                "release_corpus_regenerated"
            ]
            is False
        ),

    "div255_false":
        (
            remote_receipt[
                "scientific_boundaries"
            ][
                "float32_div255_in_timer"
            ]
            is False
        ),

    "inference_false":
        (
            remote_receipt[
                "scientific_boundaries"
            ][
                "model_inference_in_timer"
            ]
            is False
        ),

    "page_cache_not_manipulated":
        (
            remote_receipt[
                "scientific_boundaries"
            ][
                "page_cache_manipulated"
            ]
            is False
        ),

    "cold_claim_false":
        (
            remote_receipt[
                "scientific_boundaries"
            ][
                "cold_storage_claim_allowed"
            ]
            is False
        ),

    "E2E_false":
        (
            remote_receipt[
                "scientific_boundaries"
            ][
                "complete_E2E_claim_allowed"
            ]
            is False
        ),

    "GPU_false":
        (
            remote_receipt[
                "scientific_boundaries"
            ][
                "GPU_used"
            ]
            is False
        ),
}


for name, passed in remote_checks.items():

    print(
        f"{name:44s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    remote_checks.values()
):

    raise RuntimeError(
        "Remote Stage26-4C3 scientific audit failed."
    )


# =============================================================================
# 30. FINAL GIT AUDIT
# =============================================================================

banner(
    "STAGE26-4C3 :: FINAL GIT AUDIT"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != final_remote:

    raise RuntimeError(
        "Local/remote divergence after Stage26-4C3."
    )


if final_status:

    raise RuntimeError(
        "Repository not clean after Stage26-4C3."
    )


# =============================================================================
# 31. CLOSURE
# =============================================================================

banner(
    "STAGE26-4C3 CPU1 REPRESENTATION TIMING COMPLETE"
)


print(
    "NEW DURABLE COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nPARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nMEASUREMENT:"
)

print(
    "  component : compact Release corpus -> dense image + byte mask"
)

print(
    "  CPU       : CPU1"
)

print(
    "  affinity  : [0]"
)

print(
    "  threads   : 1"
)

print(
    "  batches   :",
    BATCH_SIZES
)

print(
    "  raw observations:",
    len(
        raw_observation_rows
    )
)


print(
    "\nFINAL SUMMARY:"
)


for record in condition_records:

    s = record[
        "summary"
    ]


    print(
        f"\n  B={record['batch_size']}"
    )

    print(
        f"    p50 batch latency : "
        f"{s['p50_batch_latency_seconds'] * 1000:.6f} ms"
    )

    print(
        f"    p95 batch latency : "
        f"{s['p95_batch_latency_seconds'] * 1000:.6f} ms"
    )


    if s[
        "p99_batch_latency_seconds"
    ] is None:

        print(
            "    p99 batch latency : N/A (n < 100)"
        )

    else:

        print(
            f"    p99 batch latency : "
            f"{s['p99_batch_latency_seconds'] * 1000:.6f} ms"
        )


    print(
        f"    median flows/s    : "
        f"{s['median_flows_per_second']:,.3f}"
    )

    print(
        f"    min flows/s       : "
        f"{s['minimum_flows_per_second']:,.3f}"
    )

    print(
        f"    max flows/s       : "
        f"{s['maximum_flows_per_second']:,.3f}"
    )


print(
    "\nDURABLE HASHES:"
)

print(
    "  execution plan :",
    execution_plan_sha
)

print(
    "  raw CSV        :",
    raw_csv_sha
)

print(
    "  summary JSON   :",
    summary_json_sha
)

print(
    "  summary CSV    :",
    summary_csv_sha
)

print(
    "  receipt        :",
    receipt_sha
)

print(
    "  manifest       :",
    manifest_sha
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  commit                  : PASS"
)

print(
    "  parent                  : PASS"
)

print(
    "  manifest                : PASS"
)

print(
    "  all checkpoint files    : PASS"
)

print(
    "  scientific content      : PASS"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  CPU1 REPRESENTATION TIMING COMPLETE : YES"
)

print(
    "  ALL FIVE CONDITIONS PASS            : YES"
)

print(
    "  LABELS ACCESSED                     : NO"
)

print(
    "  PCAP ACCESSED                       : NO"
)

print(
    "  CORPUS REGENERATED                  : NO"
)

print(
    "  RAW MASKING/ENCODING MEASURED       : NO"
)

print(
    "  FLOAT32 /255 INCLUDED               : NO"
)

print(
    "  MODEL INFERENCE INCLUDED            : NO"
)

print(
    "  COLD-STORAGE CLAIM                  : NO"
)

print(
    "  COMPLETE E2E CLAIM                  : NO"
)

print(
    "  GPU                                 : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Audit the completed CPU representation results and determine"
)

print(
    "  the remaining CPU-only Stage26 closure work before enabling"
)

print(
    "  the final consolidated GPU profiling phase."
)


STAGE26-4C3 :: DURABLE SCIENTIFIC PARENT
Expected parent: d6ec86b1a6fb7ce69c254dd1c6238eb3ec0f459b
Local HEAD     : d6ec86b1a6fb7ce69c254dd1c6238eb3ec0f459b
origin/main    : d6ec86b1a6fb7ce69c254dd1c6238eb3ec0f459b
Repo clean     : True

STAGE26-4C3 :: FROZEN INPUT IDENTITY
4C1 protocol                     PASS 4c446212f1af49a6c70c3ef6fcd22facb3f67c650ce6f12a4790807bd58bcb2f
4C1 boundary                     PASS 88efc94664d9658b1e0601506ac4ea369300a98a05049e79280a77107c0828ba
effective worker v2              PASS ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23
4C1A erratum                     PASS 1f6748b3042b64626de73c74f9a9ff2f8c1f9ce5e6b445a5dd2030137f951839
4C2 equivalence result           PASS d8b28e8aea4f898bf4c6d5eba95706bbffeaca4270e596ee63529d013e88da10
4C2 equivalence receipt          PASS 4dd18384f0c5d2dbc68711fbb4fd11dc7facd550a4be9db2b3ab13cc5fda2351
4C2 restoration receipt          PASS 6f31d78553a24695eb378abd65823dfaa400325557e78c1d26686d031fb8a3d2
4C2

In [10]:
# =============================================================================
# STAGE26-5A
# END-TO-END AVAILABILITY / ADDITIVITY CLOSURE
#
# DURABLE SCIENTIFIC PARENT:
#   aee59fabab2465cca7c80904279bb6d3ef23894f
#
# PURPOSE
# -------
# Determine whether the already-completed Stage26 CPU measurements can support
# a scientifically valid COMPLETE end-to-end deployment latency / throughput
# claim.
#
# THIS IS NOT A PERFORMANCE MEASUREMENT.
#
# CRITICAL RULE
# -------------
# We must NOT create an artificial:
#
#     extraction + representation + inference = complete E2E
#
# unless the output boundary of each upstream component is exactly the input
# boundary of the next component and all intervening operations have been
# accounted for.
#
# CURRENT FROZEN COMPONENTS
# -------------------------
#
# Component A:
#   raw Monday PCAPNG
#       ->
#   source-faithful IPv4/TCP/UDP/OTHER0 parsing
#       ->
#   bidirectional flow lifecycle reconstruction/count geometry
#
#   Output:
#       in-memory reconstructed flow lifecycle state/counts
#
#   NOT:
#       Stage20 compact corpus
#       70-feature CICFlowMeter matrix
#
#
# Component B:
#   existing authoritative Stage20 compact Release corpus
#       ->
#   dense uint8 [64,256] image
#   dense bool  [64,256] padding mask
#
#   The Release corpus ALREADY lies downstream of:
#       raw parsing
#       flow reconstruction
#       supervised matching/filtering
#       packet selection
#       packet truncation
#       masking/encoding
#
#
# Component C:
#   isolated model inference already measured in Stage26-2.
#
#
# CONSEQUENCE
# -----------
#
# GROUP A:
#   XGBoost / LightGBM / CatBoost / FT ensemble
#
#   Complete E2E unavailable because the deployment path requires the
#   70-feature representation/extraction boundary, which Stage26 has not
#   profiled.
#
#
# GROUP B:
#   CNN / ViT
#
#   Complete raw-PCAP -> model E2E unavailable because Component A does not
#   produce the input consumed by Component B. Missing bridge operations were
#   intentionally not measured and must not be inferred from the Release corpus.
#
#
# THEREFORE:
#
#   - no summed A+B+C latency
#   - no summed complete-E2E throughput
#   - no "full pipeline bottleneck" claim spanning missing boundaries
#   - component-level measurements remain valid
#
#
# THIS CELL:
#   1. verifies current durable parent and repo cleanliness;
#   2. verifies Stage26-2, 4B, 4C2, 4C3 artifacts exist;
#   3. audits exact frozen component boundaries;
#   4. creates a formal Group-A / Group-B E2E availability decision;
#   5. freezes the remaining CPU-only Stage26 roadmap;
#   6. commits + pushes;
#   7. remotely byte-verifies the decision package.
#
# NO:
#   - timing
#   - inference
#   - model loading
#   - PCAP access
#   - Release corpus access
#   - representation reconstruction
#   - labels
#   - GPU
# =============================================================================

from __future__ import annotations

import os
import json
import stat
import hashlib
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. PATHS / IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "aee59fabab2465cca7c80904279bb6d3ef23894f"
)

COMMIT_SUBJECT = (
    "stage26: close complete end-to-end availability"
)


ROOT = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
)


# -----------------------------------------------------------------------------
# Stage26-2 inference
# -----------------------------------------------------------------------------

WARM_DIR = (
    ROOT
    / "stage26_2_cpu_warm_inference"
)

WARM_RECEIPT = (
    WARM_DIR
    / "stage26_2_warm_cpu_receipt.json"
)

WARM_MANIFEST = (
    WARM_DIR
    / "stage26_2_warm_cpu_package_manifest.json"
)


# -----------------------------------------------------------------------------
# Stage26-4B extraction boundary + measured result
# -----------------------------------------------------------------------------

EXTRACTION_LOCK_DIR = (
    ROOT
    / "stage26_4b_extraction_protocol_lock"
)

EXTRACTION_BOUNDARY = (
    EXTRACTION_LOCK_DIR
    / "stage26_component_boundary_map.json"
)

EXTRACTION_PROTOCOL = (
    EXTRACTION_LOCK_DIR
    / "stage26_extraction_protocol.json"
)

EXTRACTION_TIMING_DIR = (
    ROOT
    / "stage26_4b3_cpu1_extraction_timing"
)


# -----------------------------------------------------------------------------
# Stage26-4C representation boundary
# -----------------------------------------------------------------------------

REP_LOCK_DIR = (
    ROOT
    / "stage26_4c_representation_protocol_lock"
)

REP_BOUNDARY = (
    REP_LOCK_DIR
    / "stage26_representation_boundary_map.json"
)

REP_PROTOCOL = (
    REP_LOCK_DIR
    / "stage26_representation_protocol.json"
)


# -----------------------------------------------------------------------------
# Stage26-4C2 equivalence
# -----------------------------------------------------------------------------

EQUIV_DIR = (
    ROOT
    / "stage26_4c2_representation_equivalence"
)

EQUIV_RECEIPT = (
    EQUIV_DIR
    / "stage26_4c2_equivalence_receipt.json"
)

EQUIV_MANIFEST = (
    EQUIV_DIR
    / "stage26_4c2_representation_equivalence_manifest.json"
)


# -----------------------------------------------------------------------------
# Stage26-4C3 measured representation
# -----------------------------------------------------------------------------

REP_TIMING_DIR = (
    ROOT
    / "stage26_4c3_cpu1_representation_timing"
)

REP_SUMMARY = (
    REP_TIMING_DIR
    / "stage26_4c3_cpu1_representation_summary.json"
)

REP_RECEIPT = (
    REP_TIMING_DIR
    / "stage26_4c3_cpu1_representation_timing_receipt.json"
)

REP_MANIFEST = (
    REP_TIMING_DIR
    / "stage26_4c3_cpu1_representation_timing_manifest.json"
)


# =============================================================================
# 1. OUTPUT CHECKPOINT
# =============================================================================

CHECKPOINT_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_5a_e2e_availability_closure"
)

CHECKPOINT_DIR = (
    REPO
    / CHECKPOINT_REL
)

INVENTORY_PATH = (
    CHECKPOINT_DIR
    / "stage26_5a_cpu_component_inventory.json"
)

DECISION_PATH = (
    CHECKPOINT_DIR
    / "stage26_5a_e2e_availability_decision.json"
)

ROADMAP_PATH = (
    CHECKPOINT_DIR
    / "stage26_5a_remaining_cpu_roadmap.json"
)

RECEIPT_PATH = (
    CHECKPOINT_DIR
    / "stage26_5a_e2e_availability_receipt.json"
)

MANIFEST_PATH = (
    CHECKPOINT_DIR
    / "stage26_5a_e2e_availability_manifest.json"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 122
    )

    print(text)

    print(
        "=" * 122
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        check=True,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        text=False,
    )

    return bytes(
        p.stdout
    )


def unique_glob(
    directory,
    pattern,
):

    matches = sorted(
        directory.glob(
            pattern
        )
    )

    if len(
        matches
    ) != 1:

        raise RuntimeError(
            f"Expected exactly one {pattern!r} in {directory}; "
            f"found {len(matches)}."
        )

    return matches[
        0
    ]


# =============================================================================
# 3. DURABLE PARENT GATE
# =============================================================================

banner(
    "STAGE26-5A :: DURABLE SCIENTIFIC PARENT"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-5A parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before E2E availability closure."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if CHECKPOINT_DIR.exists():

    raise RuntimeError(
        "Stage26-5A checkpoint already exists."
    )


# =============================================================================
# 4. REQUIRED DURABLE ARTIFACT GATE
# =============================================================================

banner(
    "STAGE26-5A :: REQUIRED DURABLE ARTIFACTS"
)


required = [
    (
        "Stage26-2 warm receipt",
        WARM_RECEIPT,
    ),

    (
        "Stage26-2 warm manifest",
        WARM_MANIFEST,
    ),

    (
        "4B extraction boundary",
        EXTRACTION_BOUNDARY,
    ),

    (
        "4B extraction protocol",
        EXTRACTION_PROTOCOL,
    ),

    (
        "4C representation boundary",
        REP_BOUNDARY,
    ),

    (
        "4C representation protocol",
        REP_PROTOCOL,
    ),

    (
        "4C2 equivalence receipt",
        EQUIV_RECEIPT,
    ),

    (
        "4C2 equivalence manifest",
        EQUIV_MANIFEST,
    ),

    (
        "4C3 representation summary",
        REP_SUMMARY,
    ),

    (
        "4C3 representation receipt",
        REP_RECEIPT,
    ),

    (
        "4C3 representation manifest",
        REP_MANIFEST,
    ),
]


artifact_hashes = {}


for label, path in required:

    if not path.is_file():

        raise FileNotFoundError(
            path
        )

    digest = sha256_file(
        path
    )

    artifact_hashes[
        str(
            path.relative_to(
                REPO
            )
        )
    ] = digest

    print(
        f"{label:34s} PASS {digest}"
    )


# Locate the unique 4B3 manifest without relying on a guessed filename.
extraction_manifest = unique_glob(
    EXTRACTION_TIMING_DIR,
    "*manifest.json",
)

extraction_manifest_sha = sha256_file(
    extraction_manifest
)


print(
    f"{'4B3 extraction manifest':34s} "
    f"PASS {extraction_manifest_sha}"
)


# =============================================================================
# 5. LOAD FROZEN SCIENTIFIC BOUNDARIES
# =============================================================================

banner(
    "STAGE26-5A :: FROZEN COMPONENT BOUNDARIES"
)


extraction_boundary = json.loads(
    EXTRACTION_BOUNDARY.read_text(
        encoding="utf-8"
    )
)

representation_boundary = json.loads(
    REP_BOUNDARY.read_text(
        encoding="utf-8"
    )
)

warm_receipt = json.loads(
    WARM_RECEIPT.read_text(
        encoding="utf-8"
    )
)

equiv_receipt = json.loads(
    EQUIV_RECEIPT.read_text(
        encoding="utf-8"
    )
)

rep_summary = json.loads(
    REP_SUMMARY.read_text(
        encoding="utf-8"
    )
)

rep_receipt = json.loads(
    REP_RECEIPT.read_text(
        encoding="utf-8"
    )
)


# =============================================================================
# 6. COMPONENT COMPLETION AUDIT
# =============================================================================

banner(
    "STAGE26-5A :: CPU COMPONENT COMPLETION AUDIT"
)


component_checks = {
    "Stage26-2 warm inference exists":
        WARM_RECEIPT.is_file(),

    "4B extraction timing exists":
        EXTRACTION_TIMING_DIR.is_dir(),

    "4C2 equivalence PASS":
        (
            equiv_receipt[
                "status"
            ]
            ==
            "PASS_EXACT_128_FLOW_REPRESENTATION_EQUIVALENCE"
        ),

    "4C3 representation PASS":
        (
            rep_summary[
                "status"
            ]
            ==
            "PASS_ALL_FIVE_FROZEN_CPU1_REPRESENTATION_CONDITIONS"
        ),

    "4C3 five conditions":
        (
            int(
                rep_summary[
                    "condition_count"
                ]
            )
            ==
            5
        ),

    "4C3 520 observations":
        (
            int(
                rep_summary[
                    "raw_observation_count"
                ]
            )
            ==
            520
        ),

    "4C3 no labels":
        (
            rep_receipt[
                "scientific_boundaries"
            ][
                "labels_accessed"
            ]
            is False
        ),

    "4C3 no PCAP":
        (
            rep_receipt[
                "scientific_boundaries"
            ][
                "pcap_accessed"
            ]
            is False
        ),

    "4C3 no GPU":
        (
            rep_receipt[
                "scientific_boundaries"
            ][
                "GPU_used"
            ]
            is False
        ),
}


for name, passed in component_checks.items():

    print(
        f"{name:42s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    component_checks.values()
):

    raise RuntimeError(
        "Completed CPU component audit failed."
    )


# =============================================================================
# 7. EXACT ADDITIVITY / COMPATIBILITY AUDIT
# =============================================================================

banner(
    "STAGE26-5A :: ADDITIVITY / E2E COMPATIBILITY AUDIT"
)


# Frozen Component A.
component_a = extraction_boundary[
    "component_A_raw_extraction"
]

component_b = extraction_boundary[
    "component_B_existing_release_corpus"
]

component_c = extraction_boundary[
    "component_C_model_inference"
]

additivity_rule = extraction_boundary[
    "additivity_rule"
]


# Independent later representation boundary.
rep_claim = representation_boundary[
    "claim_boundary"
]

rep_excluded = set(
    representation_boundary[
        "excluded_operations"
    ]
)


additivity_checks = {
    "Original A+B+C already false":
        (
            additivity_rule[
                "A_plus_B_plus_C_is_currently_complete_end_to_end"
            ]
            is False
        ),

    "A writes no corpus":
        (
            component_a[
                "corpus_written"
            ]
            is False
        ),

    "A output is lifecycle boundary":
        (
            "flow lifecycle"
            in
            component_a[
                "output_boundary"
            ].lower()
        ),

    "B is authoritative existing Release":
        (
            component_b[
                "authoritative"
            ]
            is True
        ),

    "B regeneration forbidden":
        (
            component_b[
                "regeneration_from_pcap"
            ]
            ==
            "FORBIDDEN"
        ),

    "C inference already measured":
        (
            component_c[
                "status"
            ]
            ==
            "ALREADY_MEASURED_STAGE26_2"
        ),

    "Representation complete E2E false":
        (
            rep_claim[
                "complete_end_to_end_claim_allowed"
            ]
            is False
        ),

    "Group-A 70-feature extraction absent":
        (
            rep_claim[
                "group_A_70_feature_extraction_profiled"
            ]
            is False
        ),

    "Raw-flow -> image complete cost false":
        (
            rep_claim[
                "raw_flow_to_packet_image_complete_cost"
            ]
            is False
        ),

    "Masking/encoding cost absent":
        (
            rep_claim[
                "packet_masking_encoding_cost_included"
            ]
            is False
        ),

    "Supervised matching/filtering excluded":
        (
            "supervised matching/filtering"
            in
            rep_excluded
        ),

    "Packet selection excluded":
        (
            "packet selection"
            in
            rep_excluded
        ),

    "Header masking excluded":
        (
            "header masking"
            in
            rep_excluded
        ),

    "Packet compact encoding excluded":
        (
            "packet truncation/compact encoding"
            in
            rep_excluded
        ),
}


for name, passed in additivity_checks.items():

    print(
        f"{name:48s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    additivity_checks.values()
):

    raise RuntimeError(
        "Frozen additivity audit did not support the expected boundary decision."
    )


# =============================================================================
# 8. FORMAL SCIENTIFIC DECISION
# =============================================================================

banner(
    "STAGE26-5A :: FORMAL E2E AVAILABILITY DECISION"
)


decision = {
    "schema":
        "stage26_5a_e2e_availability_decision_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-5A",

    "status":
        "CLOSED_NO_VALID_COMPLETE_E2E_MEASUREMENT",

    "scientific_parent":
        EXPECTED_PARENT,

    "decision":
        (
            "No complete end-to-end deployment latency or throughput value "
            "will be reported from the existing Stage26 CPU measurements."
        ),

    "reason":
        (
            "The measured component boundaries are not contiguous. "
            "Component A ends at source-faithful raw flow lifecycle "
            "reconstruction, while the authoritative Group-B compact corpus "
            "begins downstream of supervised matching/filtering, packet "
            "selection/truncation, and masking/encoding. Group-A deployment "
            "requires a 70-feature extraction boundary that has not been "
            "profiled. Summing measured components would therefore omit "
            "unmeasured operations and create a false complete-E2E claim."
        ),

    "group_A_dupsafe70": {
        "models": [
            "XGBoost",
            "LightGBM",
            "CatBoost",
            "FT_TRANSFORMER_5_CHECKPOINT_ENSEMBLE",
        ],

        "operational_or_reference_targets": [
            "ENS_LGBM_XGB_EQUAL",
            "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
        ],

        "complete_E2E_available":
            False,

        "blocking_gap":
            (
                "The exact deployment-time 70-feature extraction / "
                "preprocessing path feeding Group-A models has not been "
                "profiled by Stage26."
            ),

        "raw_extraction_measurement_substitutable":
            False,

        "why_raw_extraction_is_not_substitutable":
            (
                "Stage26-4B raw extraction terminates at flow lifecycle "
                "state/count geometry and does not emit the Group-A "
                "70-feature model input."
            ),
    },

    "group_B_packet_image": {
        "models": [
            "CNN",
            "ViT",
        ],

        "complete_raw_pcap_to_model_E2E_available":
            False,

        "blocking_gap": [
            "supervised matching/filtering",
            "packet selection",
            "packet truncation",
            "header masking",
            "compact packet encoding",
        ],

        "component_A_output_equals_component_B_input":
            False,

        "component_B_measurement_valid":
            True,

        "component_B_claim":
            (
                "Warm existing-compact-corpus dense materialization into "
                "uint8 [B,64,256] images and bool [B,64,256] masks."
            ),
    },

    "prohibited_derivations": {
        "sum_A_plus_B_plus_C_latency":
            True,

        "report_sum_as_complete_E2E":
            True,

        "complete_pipeline_throughput_from_component_minimum":
            True,

        "full_pipeline_bottleneck_across_missing_boundaries":
            True,

        "cross_group_extraction_equivalence":
            True,
    },

    "still_valid": {
        "Stage26_1_cold_start":
            True,

        "Stage26_2_isolated_warm_inference":
            True,

        "Stage26_3_memory":
            True,

        "Stage26_4B_raw_extraction_component":
            True,

        "Stage26_4C_compact_corpus_representation_component":
            True,

        "component_level_comparisons_with_explicit_boundaries":
            True,
    },

    "complete_E2E_status":
        "NOT_AVAILABLE_BY_DESIGN_AND_BOUNDARY",

    "missing_measurement_is_not_imputed":
        True,

    "no_new_measurement_required_for_stage26_5_closure":
        True,
}


# =============================================================================
# 9. CPU COMPONENT INVENTORY
# =============================================================================

inventory = {
    "schema":
        "stage26_5a_cpu_component_inventory_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "components": {
        "cold_start":
            "COMPLETE",

        "warm_CPU_inference":
            "COMPLETE",

        "CPU_memory":
            "COMPLETE",

        "raw_extraction_CPU1":
            "COMPLETE_COMPONENT_ONLY",

        "packet_image_representation_CPU1":
            "COMPLETE_COMPONENT_ONLY",

        "complete_E2E":
            "CLOSED_NOT_AVAILABLE",
    },

    "representation_CPU1_summary": {
        str(
            record[
                "batch_size"
            ]
        ): {
            "p50_batch_latency_seconds":
                record[
                    "summary"
                ][
                    "p50_batch_latency_seconds"
                ],

            "p95_batch_latency_seconds":
                record[
                    "summary"
                ][
                    "p95_batch_latency_seconds"
                ],

            "p99_batch_latency_seconds":
                record[
                    "summary"
                ][
                    "p99_batch_latency_seconds"
                ],

            "median_flows_per_second":
                record[
                    "summary"
                ][
                    "median_flows_per_second"
                ],
        }
        for record in rep_summary[
            "conditions"
        ]
    },

    "provenance_hashes": {
        **artifact_hashes,

        str(
            extraction_manifest.relative_to(
                REPO
            )
        ):
            extraction_manifest_sha,
    },

    "measurements_performed_by_this_checkpoint":
        False,
}


# =============================================================================
# 10. REMAINING CPU ROADMAP
# =============================================================================

roadmap = {
    "schema":
        "stage26_5a_remaining_cpu_roadmap_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "current_status": {
        "Stage26_5_complete_E2E":
            "CLOSED_NOT_AVAILABLE",

        "GPU_enabled":
            False,
    },

    "remaining_CPU_only_work": [
        {
            "stage":
                "STAGE26-6",

            "name":
                "CPU Pareto analysis",

            "required":
                True,

            "rules": [
                (
                    "GROUP_A_DUPSAFE70 Pareto only within Group A using "
                    "frozen predictive metric and CPU1 B1 inference cost."
                ),

                (
                    "GROUP_B_PACKET_IMAGE CNN/ViT comparison only within "
                    "Group B and explicitly descriptive/non-confirmatory."
                ),

                "No cross-group Pareto dominance.",

                (
                    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE excluded from "
                    "predictive Pareto."
                ),

                (
                    "ENS_LGBM_XGB_EQUAL treated as operational/reference "
                    "according to frozen protocol."
                ),
            ],
        },

        {
            "stage":
                "STAGE26-7",

            "name":
                "CPU batch scaling and component bottleneck analysis",

            "required":
                True,

            "rules": [
                (
                    "Use only scientifically compatible component boundaries."
                ),

                (
                    "Do not label component ratios as complete-pipeline "
                    "bottlenecks where missing stages exist."
                ),

                (
                    "Preserve timeout/OOM results as frozen resource limits."
                ),
            ],
        },

        {
            "stage":
                "STAGE26-8",

            "name":
                "CPU audit, figures, tables, and CPU-phase closure",

            "required":
                True,

            "rules": [
                "Audit hashes and frozen protocol adherence.",

                "Generate publication-ready CPU deployment figures/tables.",

                "Persist and remotely verify final CPU closure checkpoint.",
            ],
        },
    ],

    "GPU_activation_rule": {
        "allowed_now":
            False,

        "allowed_only_after": [
            "STAGE26-6 CPU outputs committed and pushed",
            "STAGE26-7 CPU outputs committed and pushed",
            "STAGE26-8 CPU closure committed and pushed",
        ],

        "fresh_GPU_session_required":
            True,

        "GPU_phase_anchor":
            "FINAL_CPU_CLOSURE_COMMIT_NOT_YET_CREATED",
    },
}


# =============================================================================
# 11. WRITE CHECKPOINT PACKAGE
# =============================================================================

banner(
    "STAGE26-5A :: WRITE CLOSURE PACKAGE"
)


CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


atomic_json(
    INVENTORY_PATH,
    inventory,
)

atomic_json(
    DECISION_PATH,
    decision,
)

atomic_json(
    ROADMAP_PATH,
    roadmap,
)


inventory_sha = sha256_file(
    INVENTORY_PATH
)

decision_sha = sha256_file(
    DECISION_PATH
)

roadmap_sha = sha256_file(
    ROADMAP_PATH
)


receipt = {
    "schema":
        "stage26_5a_e2e_availability_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-5A",

    "status":
        "PASS_E2E_AVAILABILITY_CLOSED_WITHOUT_INVALID_ADDITIVITY",

    "scientific_parent":
        EXPECTED_PARENT,

    "inventory_sha256":
        inventory_sha,

    "decision_sha256":
        decision_sha,

    "roadmap_sha256":
        roadmap_sha,

    "complete_E2E_measurement_available":
        False,

    "complete_E2E_measurement_performed":
        False,

    "complete_E2E_value_imputed":
        False,

    "component_measurements_preserved":
        True,

    "group_A_70_feature_extraction_profiled":
        False,

    "group_B_missing_raw_to_compact_bridge":
        True,

    "GPU_allowed_after_this_checkpoint":
        False,

    "remaining_CPU_stages": [
        "STAGE26-6",
        "STAGE26-7",
        "STAGE26-8",
    ],

    "scientific_state": {
        "timing_performed":
            False,

        "pcap_accessed":
            False,

        "release_corpus_accessed":
            False,

        "representation_materialized":
            False,

        "labels_accessed":
            False,

        "models_loaded":
            False,

        "inference_performed":
            False,

        "GPU_used":
            False,
    },
}


atomic_json(
    RECEIPT_PATH,
    receipt,
)


receipt_sha = sha256_file(
    RECEIPT_PATH
)


# =============================================================================
# 12. MANIFEST
# =============================================================================

package_files = [
    INVENTORY_PATH,
    DECISION_PATH,
    ROADMAP_PATH,
    RECEIPT_PATH,
]


manifest_rows = []


for path in package_files:

    manifest_rows.append(
        {
            "repo_relative_path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest = {
    "schema":
        "stage26_5a_e2e_availability_manifest_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        "READY_FOR_GIT_ANCHOR",

    "parent_commit":
        EXPECTED_PARENT,

    "commit_subject":
        COMMIT_SUBJECT,

    "inventory_sha256":
        inventory_sha,

    "decision_sha256":
        decision_sha,

    "roadmap_sha256":
        roadmap_sha,

    "receipt_sha256":
        receipt_sha,

    "file_count_excluding_manifest":
        len(
            manifest_rows
        ),

    "files":
        manifest_rows,

    "scientific_boundaries": {
        "complete_E2E_available":
            False,

        "invalid_additivity_rejected":
            True,

        "missing_boundaries_imputed":
            False,

        "new_measurement_performed":
            False,

        "GPU_used":
            False,
    },
}


atomic_json(
    MANIFEST_PATH,
    manifest,
)


manifest_sha = sha256_file(
    MANIFEST_PATH
)


print(
    "Inventory:",
    inventory_sha
)

print(
    "Decision :",
    decision_sha
)

print(
    "Roadmap  :",
    roadmap_sha
)

print(
    "Receipt  :",
    receipt_sha
)

print(
    "Manifest :",
    manifest_sha
)


# =============================================================================
# 13. PRINT SCIENTIFIC DECISION
# =============================================================================

banner(
    "STAGE26-5A :: SCIENTIFIC DECISION"
)


print(
    "GROUP A complete E2E : NOT AVAILABLE"
)

print(
    "  reason: 70-feature extraction path not profiled"
)


print(
    "\nGROUP B complete E2E : NOT AVAILABLE"
)

print(
    "  reason: raw extraction -> compact corpus bridge is not measured"
)


print(
    "\nA + B + C additive E2E: PROHIBITED"
)

print(
    "Missing operations will NOT be imputed."
)


print(
    "\nVALID COMPONENT RESULTS REMAIN:"
)

print(
    "  cold-start              : YES"
)

print(
    "  isolated inference      : YES"
)

print(
    "  CPU memory              : YES"
)

print(
    "  raw extraction          : YES, component-only"
)

print(
    "  representation          : YES, component-only"
)


print(
    "\nGPU ENABLED NOW: NO"
)

print(
    "Remaining CPU stages: 26-6, 26-7, 26-8"
)


# =============================================================================
# 14. LOCAL PACKAGE AUDIT
# =============================================================================

banner(
    "STAGE26-5A :: LOCAL PACKAGE AUDIT"
)


for row in manifest_rows:

    path = (
        REPO
        / row[
            "repo_relative_path"
        ]
    )

    actual_size = int(
        path.stat().st_size
    )

    actual_sha = sha256_file(
        path
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Local Stage26-5A package audit failed."
        )


# =============================================================================
# 15. GIT CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-5A :: GIT CHANGE AUDIT"
)


repo_status = git(
    "status",
    "--porcelain",
)


print(
    repo_status
)


if not repo_status:

    raise RuntimeError(
        "Expected uncommitted Stage26-5A package."
    )


unexpected = []


for line in repo_status.splitlines():

    rel = line[
        3:
    ]


    if not rel.startswith(
        str(
            CHECKPOINT_REL
        )
        +
        "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository changes:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 16. GIT IDENTITY
# =============================================================================

author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_PARENT,
)

author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_PARENT,
)


git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


print(
    "\nGit author:"
)

print(
    " ",
    author_name,
    "<" + author_email + ">"
)


# =============================================================================
# 17. COMMIT
# =============================================================================

banner(
    "STAGE26-5A :: COMMIT"
)


git(
    "add",
    str(
        CHECKPOINT_REL
    ),
)


print(
    git(
        "diff",
        "--cached",
        "--name-status",
    )
)


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-5A commit parent mismatch."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Stage26-5A commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository not clean after commit."
    )


# =============================================================================
# 18. PUSH
# =============================================================================

banner(
    "STAGE26-5A :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    pushed = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
        text=True,
    )


    print(
        pushed.stdout.strip()
    )


github_token = None


# =============================================================================
# 19. REMOTE COMMIT + BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-5A :: REMOTE VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "origin/main != Stage26-5A commit."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote Stage26-5A parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote subject mismatch."
    )


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    str(
        MANIFEST_PATH.relative_to(
            REPO
        )
    ),
)

remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "\nRemote manifest SHA256:"
)

print(
    " ",
    remote_manifest_sha
)


if remote_manifest_sha != manifest_sha:

    raise RuntimeError(
        "Remote manifest mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


for row in remote_manifest[
    "files"
]:

    data = git_blob_bytes(
        "origin/main",
        row[
            "repo_relative_path"
        ],
    )

    actual_size = len(
        data
    )

    actual_sha = sha256_bytes(
        data
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Remote Stage26-5A byte verification failed."
        )


# =============================================================================
# 20. REMOTE SCIENTIFIC AUDIT
# =============================================================================

banner(
    "STAGE26-5A :: REMOTE SCIENTIFIC AUDIT"
)


remote_decision = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            DECISION_PATH.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_roadmap = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            ROADMAP_PATH.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_receipt = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            RECEIPT_PATH.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)


remote_checks = {
    "Stage26-5 closed":
        (
            remote_decision[
                "status"
            ]
            ==
            "CLOSED_NO_VALID_COMPLETE_E2E_MEASUREMENT"
        ),

    "complete E2E false":
        (
            remote_decision[
                "complete_E2E_status"
            ]
            ==
            "NOT_AVAILABLE_BY_DESIGN_AND_BOUNDARY"
        ),

    "Group A E2E false":
        (
            remote_decision[
                "group_A_dupsafe70"
            ][
                "complete_E2E_available"
            ]
            is False
        ),

    "Group B E2E false":
        (
            remote_decision[
                "group_B_packet_image"
            ][
                "complete_raw_pcap_to_model_E2E_available"
            ]
            is False
        ),

    "missing values not imputed":
        (
            remote_decision[
                "missing_measurement_is_not_imputed"
            ]
            is True
        ),

    "GPU still false":
        (
            remote_roadmap[
                "GPU_activation_rule"
            ][
                "allowed_now"
            ]
            is False
        ),

    "remaining 26-6/7/8":
        (
            remote_receipt[
                "remaining_CPU_stages"
            ]
            ==
            [
                "STAGE26-6",
                "STAGE26-7",
                "STAGE26-8",
            ]
        ),

    "no new timing":
        (
            remote_receipt[
                "scientific_state"
            ][
                "timing_performed"
            ]
            is False
        ),

    "no GPU":
        (
            remote_receipt[
                "scientific_state"
            ][
                "GPU_used"
            ]
            is False
        ),
}


for name, passed in remote_checks.items():

    print(
        f"{name:40s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    remote_checks.values()
):

    raise RuntimeError(
        "Remote Stage26-5A scientific audit failed."
    )


# =============================================================================
# 21. FINAL AUDIT / CLOSURE
# =============================================================================

banner(
    "STAGE26-5A END-TO-END AVAILABILITY CLOSURE COMPLETE"
)


final_status = git(
    "status",
    "--porcelain",
)


print(
    "NEW DURABLE COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nPARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nEND-TO-END DECISION:"
)

print(
    "  GROUP A complete E2E : NOT AVAILABLE"
)

print(
    "  GROUP B complete E2E : NOT AVAILABLE"
)

print(
    "  A+B+C summation      : PROHIBITED"
)

print(
    "  missing costs imputed: NO"
)


print(
    "\nSTAGE26-5 STATUS:"
)

print(
    "  CLOSED_NO_VALID_COMPLETE_E2E_MEASUREMENT"
)


print(
    "\nREMAINING CPU WORK:"
)

print(
    "  Stage26-6 : CPU Pareto analysis"
)

print(
    "  Stage26-7 : CPU batch/component bottleneck analysis"
)

print(
    "  Stage26-8 : CPU audit + publication figures + closure"
)


print(
    "\nGPU:"
)

print(
    "  ENABLED NOW : NO"
)

print(
    "  Enable only after Stage26-8 final CPU closure is pushed."
)


print(
    "\nHASHES:"
)

print(
    "  inventory :",
    inventory_sha
)

print(
    "  decision  :",
    decision_sha
)

print(
    "  roadmap   :",
    roadmap_sha
)

print(
    "  receipt   :",
    receipt_sha
)

print(
    "  manifest  :",
    manifest_sha
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  commit             : PASS"
)

print(
    "  parent             : PASS"
)

print(
    "  package bytes      : PASS"
)

print(
    "  scientific content : PASS"
)


print(
    "\nRepo clean:",
    final_status == ""
)


if final_status:

    raise RuntimeError(
        "Repository not clean after Stage26-5A."
    )


print(
    "\nNEXT:"
)

print(
    "  Stage26-6 CPU Pareto audit."
)

print(
    "  Before computing dominance, resolve the exact frozen predictive"
)

print(
    "  metric artifacts and CPU1 B=1 inference-cost field for each target."
)

print(
    "  No GPU yet."
)


STAGE26-5A :: DURABLE SCIENTIFIC PARENT
Expected parent: aee59fabab2465cca7c80904279bb6d3ef23894f
Local HEAD     : aee59fabab2465cca7c80904279bb6d3ef23894f
origin/main    : aee59fabab2465cca7c80904279bb6d3ef23894f
Repo clean     : True

STAGE26-5A :: REQUIRED DURABLE ARTIFACTS
Stage26-2 warm receipt             PASS b4b2623eabde7dd6b9acc250357a9f1ad61fa342c1dd496dcb634d4bfaca4d15
Stage26-2 warm manifest            PASS 7f4d01bb3fe685dca528a4a1c788bc5819b42a4e3080098fc3e923b2fb0e0f19
4B extraction boundary             PASS 3f798bf9665dabe245e627b3a21ae0ecd2f837f84b091ad07bc13f60f97458cf
4B extraction protocol             PASS 12f7f8f0332d92694a827e3473dc1cef66939cf1cf1c49c17ad1699e60d929b0
4C representation boundary         PASS 88efc94664d9658b1e0601506ac4ea369300a98a05049e79280a77107c0828ba
4C representation protocol         PASS 4c446212f1af49a6c70c3ef6fcd22facb3f67c650ce6f12a4790807bd58bcb2f
4C2 equivalence receipt            PASS 4dd18384f0c5d2dbc68711fbb4fd11dc7facd550a4be9db2b3a

In [11]:
# =============================================================================
# STAGE26-6A
# PARETO SOURCE-RESOLUTION INVENTORY
#
# DURABLE SCIENTIFIC PARENT:
#   92495eac2c9202973ff4d889e3c12669c06e9ef1
#
# PURPOSE
# -------
# Resolve, WITHOUT computing Pareto dominance:
#
#   A. exact frozen predictive PR-AUC artifact(s)
#   B. exact Stage26-2 CPU1 B=1 p95 inference-latency field(s)
#
# for the frozen Pareto-eligible targets.
#
# FROZEN GROUP A:
#   STAGE16_XGBOOST_TUNED
#   STAGE16_LIGHTGBM_TUNED
#   STAGE16_CATBOOST_TUNED
#   FT_BALANCED_5_CHECKPOINT_SOFT_VOTING
#
# FROZEN GROUP B:
#   STAGE20_MASKED_CNN_V1
#   STAGE21_MASKED_VIT_V1
#
# OPERATIONAL / RESOURCE REFERENCES:
#   ENS_LGBM_XGB_EQUAL
#   FT_BALANCED_SINGLE_RESOURCE_REFERENCE
#
# ABSOLUTE RULES
# --------------
#   - NO Pareto dominance calculation.
#   - NO metric recomputation.
#   - NO holdout reopening.
#   - NO model loading.
#   - NO inference.
#   - NO timing.
#   - NO PCAP.
#   - NO Release corpus.
#   - NO GPU.
#   - NO Git modification.
#
# This cell only inventories ALREADY-COMMITTED textual result artifacts.
#
# It writes one TRANSIENT inventory outside the Git repository:
#
#   /kaggle/working/stage26_deployment_profiling/pareto/
#       stage26_6a_source_resolution_inventory.json
#
# We inspect its output before freezing Stage26-6 inputs.
# =============================================================================

from __future__ import annotations

import csv
import io
import os
import re
import json
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. PATHS / IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "92495eac2c9202973ff4d889e3c12669c06e9ef1"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

RUNTIME_OUT_DIR = (
    STAGE26_ROOT
    / "pareto"
)

RUNTIME_OUT = (
    RUNTIME_OUT_DIR
    / "stage26_6a_source_resolution_inventory.json"
)


PROTOCOL = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)


WARM_DIR_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_2_cpu_warm_inference/"
)


# =============================================================================
# 1. TARGETS
# =============================================================================

PRIMARY_TARGETS = {
    "STAGE16_XGBOOST_TUNED": {
        "group":
            "GROUP_A_DUPSAFE70",

        "aliases": [
            "STAGE16_XGBOOST_TUNED",
            "XGBOOST",
            "XGBoost",
            "XGB",
        ],
    },

    "STAGE16_LIGHTGBM_TUNED": {
        "group":
            "GROUP_A_DUPSAFE70",

        "aliases": [
            "STAGE16_LIGHTGBM_TUNED",
            "LIGHTGBM",
            "LightGBM",
            "LGBM",
        ],
    },

    "STAGE16_CATBOOST_TUNED": {
        "group":
            "GROUP_A_DUPSAFE70",

        "aliases": [
            "STAGE16_CATBOOST_TUNED",
            "CATBOOST",
            "CatBoost",
        ],
    },

    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING": {
        "group":
            "GROUP_A_DUPSAFE70",

        "aliases": [
            "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
            "5_CHECKPOINT_SOFT_VOTING",
            "5-checkpoint",
            "soft_voting",
            "soft voting",
            "FT ensemble",
            "FT_TRANSFORMER_ENSEMBLE",
        ],
    },

    "STAGE20_MASKED_CNN_V1": {
        "group":
            "GROUP_B_PACKET_IMAGE",

        "aliases": [
            "STAGE20_MASKED_CNN_V1",
            "MASKED_CNN",
            "Masked CNN",
            "CNN",
        ],
    },

    "STAGE21_MASKED_VIT_V1": {
        "group":
            "GROUP_B_PACKET_IMAGE",

        "aliases": [
            "STAGE21_MASKED_VIT_V1",
            "MASKED_VIT",
            "Masked ViT",
            "ViT",
            "VISION_TRANSFORMER",
        ],
    },
}


REFERENCE_TARGETS = {
    "ENS_LGBM_XGB_EQUAL": {
        "role":
            "OPERATIONAL_REFERENCE",

        "aliases": [
            "ENS_LGBM_XGB_EQUAL",
            "LGBM_XGB_EQUAL",
            "equal ensemble",
        ],
    },

    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE": {
        "role":
            "RESOURCE_ONLY_EXCLUDED_FROM_PREDICTIVE_PARETO",

        "aliases": [
            "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
            "FT single",
            "seed7",
            "seed 7",
        ],
    },
}


# =============================================================================
# 2. SEARCH SCOPES
# =============================================================================

# Predictive metrics must come from frozen historical result artifacts,
# not Stage26 timing inputs.
PREDICTIVE_PREFIXES = [
    "results/stage15",
    "results/stage16",
    "results/stage17",
    "results/stage18",
    "results/stage19",
    "results/stage20",
    "results/stage21",
]


TEXT_SUFFIXES = {
    ".json",
    ".jsonl",
    ".csv",
    ".md",
    ".txt",
    ".yaml",
    ".yml",
}


# Avoid giant unrelated textual artifacts.
MAX_TEXT_BYTES = (
    8
    *
    1024
    *
    1024
)


PR_METRIC_REGEX = re.compile(
    r"(?i)"
    r"(?:"
    r"pr[\s_\-]*auc"
    r"|auc[\s_\-]*pr"
    r"|auprc"
    r"|average[\s_\-]*precision"
    r")"
)


P95_REGEX = re.compile(
    r"(?i)"
    r"(?:p95|95(?:th)?[\s_\-]*percentile)"
)


LATENCY_REGEX = re.compile(
    r"(?i)"
    r"(?:latency|elapsed|duration|inference)"
)


POPULATION_REGEX = re.compile(
    r"(?i)"
    r"(?:"
    r"population"
    r"|dupsafe"
    r"|dup[\s_\-]*safe"
    r"|friday"
    r"|holdout"
    r"|test"
    r"|validation"
    r")"
)


# =============================================================================
# 3. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        +
        "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            p.stdout
        )

    return p.stdout.strip()


def git(
    *args,
):

    return run(
        [
            "git",
            *args,
        ]
    )


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8
                *
                1024
                *
                1024
            )

            if not block:

                break

            h.update(
                block
            )

    return h.hexdigest()


def safe_text(path):

    path = Path(
        path
    )

    if not path.is_file():

        return None

    if path.suffix.lower() not in TEXT_SUFFIXES:

        return None

    if path.stat().st_size > MAX_TEXT_BYTES:

        return None

    try:

        return path.read_text(
            encoding="utf-8",
            errors="replace",
        )

    except Exception:

        return None


def normalize(value):

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(
            value
        ).lower(),
    )


def alias_hit(
    text,
    aliases,
):

    text_lower = text.lower()

    hits = []

    for alias in aliases:

        if alias.lower() in text_lower:

            hits.append(
                alias
            )

    return hits


def json_walk(
    obj,
    path="$",
):

    if isinstance(
        obj,
        dict,
    ):

        yield (
            path,
            obj,
        )

        for key, value in obj.items():

            child = (
                path
                +
                "."
                +
                str(
                    key
                )
            )

            yield from json_walk(
                value,
                child,
            )


    elif isinstance(
        obj,
        list,
    ):

        for index, value in enumerate(
            obj
        ):

            child = (
                path
                +
                f"[{index}]"
            )

            yield from json_walk(
                value,
                child,
            )


def scalar_metric_pairs(
    obj,
):

    pairs = []

    if not isinstance(
        obj,
        dict,
    ):

        return pairs

    for key, value in obj.items():

        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            )
        ) or value is None:

            pairs.append(
                (
                    str(
                        key
                    ),
                    value,
                )
            )

    return pairs


def compact_dict(
    obj,
    max_items=18,
):

    if not isinstance(
        obj,
        dict,
    ):

        return obj

    result = {}

    for index, (
        key,
        value,
    ) in enumerate(
        obj.items()
    ):

        if index >= max_items:

            result[
                "...truncated..."
            ] = (
                len(
                    obj
                )
                -
                max_items
            )

            break

        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            )
        ) or value is None:

            result[
                str(
                    key
                )
            ] = value

    return result


def line_evidence(
    text,
    aliases,
    require_regex=None,
    max_lines=8,
):

    evidence = []

    for line_number, line in enumerate(
        text.splitlines(),
        start=1,
    ):

        if not alias_hit(
            line,
            aliases,
        ):

            continue

        if (
            require_regex is not None
            and
            not require_regex.search(
                line
            )
        ):

            continue

        evidence.append(
            {
                "line":
                    line_number,

                "text":
                    line[
                        :1000
                    ],
            }
        )

        if len(
            evidence
        ) >= max_lines:

            break

    return evidence


# =============================================================================
# 4. DURABLE GIT GATE
# =============================================================================

banner(
    "STAGE26-6A :: DURABLE GIT GATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-6A parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Pareto source resolution."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


# =============================================================================
# 5. PROTOCOL IDENTITY / FROZEN PARETO RULES
# =============================================================================

banner(
    "STAGE26-6A :: FROZEN PARETO PROTOCOL"
)


protocol_sha = sha256_file(
    PROTOCOL
)


print(
    "Protocol SHA256:"
)

print(
    " expected:",
    EXPECTED_PROTOCOL_SHA256
)

print(
    " actual  :",
    protocol_sha
)


if protocol_sha != EXPECTED_PROTOCOL_SHA256:

    raise RuntimeError(
        "Stage26 measurement protocol identity mismatch."
    )


protocol = json.loads(
    PROTOCOL.read_text(
        encoding="utf-8"
    )
)


pareto = protocol[
    "pareto_protocol"
]


print(
    "\nGROUP A:"
)

print(
    json.dumps(
        pareto[
            "GROUP_A_DUPSAFE70"
        ],
        indent=2,
        sort_keys=True,
    )
)


print(
    "\nGROUP B:"
)

print(
    json.dumps(
        pareto[
            "GROUP_B_PACKET_IMAGE"
        ],
        indent=2,
        sort_keys=True,
    )
)


print(
    "\nDOMINANCE:"
)

print(
    json.dumps(
        pareto[
            "dominance_rule"
        ],
        indent=2,
        sort_keys=True,
    )
)


print(
    "\nGLOBAL RULE:"
)

print(
    pareto[
        "global_rule"
    ]
)


print(
    "\nOPERATIONAL REFERENCES:"
)

print(
    json.dumps(
        pareto[
            "operational_references"
        ],
        indent=2,
        sort_keys=True,
    )
)


# Verify our local target universe exactly matches frozen primary members.
frozen_group_a = pareto[
    "GROUP_A_DUPSAFE70"
][
    "eligible_primary_members"
]

frozen_group_b = pareto[
    "GROUP_B_PACKET_IMAGE"
][
    "eligible_primary_members"
]


if frozen_group_a != [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
]:

    raise RuntimeError(
        "Frozen Group-A Pareto universe changed."
    )


if frozen_group_b != [
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
]:

    raise RuntimeError(
        "Frozen Group-B Pareto universe changed."
    )


# =============================================================================
# 6. COMMITTED TEXT-FILE INVENTORY
# =============================================================================

banner(
    "STAGE26-6A :: COMMITTED TEXT ARTIFACT INVENTORY"
)


tracked_paths = [
    line.strip()
    for line in git(
        "ls-files"
    ).splitlines()
    if line.strip()
]


predictive_paths = []

warm_paths = []


for rel in tracked_paths:

    path = (
        REPO
        /
        rel
    )


    if path.suffix.lower() not in TEXT_SUFFIXES:

        continue


    if (
        path.exists()
        and
        path.stat().st_size
        <=
        MAX_TEXT_BYTES
    ):

        if any(
            rel.startswith(
                prefix
            )
            for prefix in PREDICTIVE_PREFIXES
        ):

            predictive_paths.append(
                rel
            )


        if rel.startswith(
            WARM_DIR_REL
        ):

            warm_paths.append(
                rel
            )


print(
    "Predictive-scope text artifacts:",
    len(
        predictive_paths
    )
)

print(
    "Stage26-2 text artifacts        :",
    len(
        warm_paths
    )
)


# =============================================================================
# 7. PREDICTIVE PR-AUC ARTIFACT DISCOVERY
# =============================================================================

banner(
    "STAGE26-6A :: PREDICTIVE PR-AUC SOURCE CANDIDATES"
)


predictive_candidates = {
    target_id: []
    for target_id in PRIMARY_TARGETS
}


for rel in predictive_paths:

    path = (
        REPO
        /
        rel
    )

    text = safe_text(
        path
    )


    if text is None:

        continue


    # PR metric evidence must exist somewhere in the artifact.
    if not PR_METRIC_REGEX.search(
        text
    ):

        continue


    file_sha = sha256_file(
        path
    )


    for target_id, metadata in PRIMARY_TARGETS.items():

        aliases = metadata[
            "aliases"
        ]

        hits = alias_hit(
            text,
            aliases,
        )


        if not hits:

            continue


        record = {
            "repo_relative_path":
                rel,

            "sha256":
                file_sha,

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "aliases_matched":
                sorted(
                    set(
                        hits
                    )
                ),

            "line_evidence":
                line_evidence(
                    text,
                    aliases,
                    require_regex=PR_METRIC_REGEX,
                    max_lines=8,
                ),

            "population_evidence":
                [],
            
            "structured_metric_evidence":
                [],
        }


        # -------------------------------------------------------------
        # Population/comparability text evidence
        # -------------------------------------------------------------

        for line_number, line in enumerate(
            text.splitlines(),
            start=1,
        ):

            if not POPULATION_REGEX.search(
                line
            ):

                continue

            record[
                "population_evidence"
            ].append(
                {
                    "line":
                        line_number,

                    "text":
                        line[
                            :1000
                        ],
                }
            )

            if len(
                record[
                    "population_evidence"
                ]
            ) >= 6:

                break


        # -------------------------------------------------------------
        # JSON structured evidence
        # -------------------------------------------------------------

        if path.suffix.lower() == ".json":

            try:

                obj = json.loads(
                    text
                )

            except Exception:

                obj = None


            if obj is not None:

                for node_path, node in json_walk(
                    obj
                ):

                    if not isinstance(
                        node,
                        dict,
                    ):

                        continue


                    node_text = json.dumps(
                        node,
                        default=str,
                    )


                    if not alias_hit(
                        node_text,
                        aliases,
                    ):

                        continue


                    metric_pairs = [
                        (
                            key,
                            value,
                        )
                        for key, value in scalar_metric_pairs(
                            node
                        )
                        if PR_METRIC_REGEX.search(
                            key
                        )
                    ]


                    if not metric_pairs:

                        continue


                    record[
                        "structured_metric_evidence"
                    ].append(
                        {
                            "json_path":
                                node_path,

                            "metric_pairs":
                                metric_pairs,

                            "context":
                                compact_dict(
                                    node
                                ),
                        }
                    )


                    if len(
                        record[
                            "structured_metric_evidence"
                        ]
                    ) >= 12:

                        break


        # -------------------------------------------------------------
        # CSV structured evidence
        # -------------------------------------------------------------

        elif path.suffix.lower() == ".csv":

            try:

                reader = csv.DictReader(
                    io.StringIO(
                        text
                    )
                )

                fieldnames = (
                    reader.fieldnames
                    or
                    []
                )

                metric_columns = [
                    column
                    for column in fieldnames
                    if PR_METRIC_REGEX.search(
                        str(
                            column
                        )
                    )
                ]


                if metric_columns:

                    for row_number, row in enumerate(
                        reader,
                        start=2,
                    ):

                        row_text = " | ".join(
                            f"{k}={v}"
                            for k, v in row.items()
                        )


                        if not alias_hit(
                            row_text,
                            aliases,
                        ):

                            continue


                        record[
                            "structured_metric_evidence"
                        ].append(
                            {
                                "csv_row":
                                    row_number,

                                "metric_pairs": [
                                    (
                                        column,
                                        row.get(
                                            column
                                        ),
                                    )
                                    for column in metric_columns
                                ],

                                "context": {
                                    key:
                                        value
                                    for key, value in row.items()
                                    if (
                                        value not in {
                                            None,
                                            "",
                                        }
                                        and
                                        (
                                            PR_METRIC_REGEX.search(
                                                str(
                                                    key
                                                )
                                            )
                                            or
                                            re.search(
                                                r"(?i)"
                                                r"(model|target|name|family|"
                                                r"split|population|day|seed|"
                                                r"ensemble|threshold)",
                                                str(
                                                    key
                                                ),
                                            )
                                        )
                                    )
                                },
                            }
                        )


                        if len(
                            record[
                                "structured_metric_evidence"
                            ]
                        ) >= 12:

                            break

            except Exception:

                pass


        predictive_candidates[
            target_id
        ].append(
            record
        )


# Sort likely structured sources before line-only sources.
for target_id in predictive_candidates:

    predictive_candidates[
        target_id
    ].sort(
        key=lambda row: (
            -len(
                row[
                    "structured_metric_evidence"
                ]
            ),
            row[
                "repo_relative_path"
            ],
        )
    )


for target_id, candidates in predictive_candidates.items():

    print(
        "\n"
        +
        "-" * 124
    )

    print(
        target_id
    )

    print(
        "Group:",
        PRIMARY_TARGETS[
            target_id
        ][
            "group"
        ]
    )

    print(
        "Candidate artifacts:",
        len(
            candidates
        )
    )


    if not candidates:

        print(
            "  [NONE FOUND]"
        )

        continue


    for candidate_index, candidate in enumerate(
        candidates[
            :12
        ],
        start=1,
    ):

        print(
            f"\n  [{candidate_index}] "
            f"{candidate['repo_relative_path']}"
        )

        print(
            "      SHA256:",
            candidate[
                "sha256"
            ]
        )

        print(
            "      aliases:",
            candidate[
                "aliases_matched"
            ]
        )


        structured = candidate[
            "structured_metric_evidence"
        ]


        if structured:

            print(
                "      STRUCTURED PR-AUC EVIDENCE:"
            )

            for evidence in structured[
                :6
            ]:

                print(
                    "       ",
                    json.dumps(
                        evidence,
                        sort_keys=True,
                        default=str,
                    )[
                        :2000
                    ]
                )


        elif candidate[
            "line_evidence"
        ]:

            print(
                "      LINE PR-AUC EVIDENCE:"
            )

            for evidence in candidate[
                "line_evidence"
            ][
                :6
            ]:

                print(
                    f"        L{evidence['line']}: "
                    f"{evidence['text']}"
                )


        if candidate[
            "population_evidence"
        ]:

            print(
                "      POPULATION/SPLIT EVIDENCE:"
            )

            for evidence in candidate[
                "population_evidence"
            ][
                :3
            ]:

                print(
                    f"        L{evidence['line']}: "
                    f"{evidence['text']}"
                )


# =============================================================================
# 8. STAGE26-2 CPU1 B=1 P95 COST SOURCE DISCOVERY
# =============================================================================

banner(
    "STAGE26-6A :: CPU1 B=1 P95 INFERENCE-COST SOURCE CANDIDATES"
)


cost_candidates = {
    target_id: []
    for target_id in PRIMARY_TARGETS
}


for rel in warm_paths:

    path = (
        REPO
        /
        rel
    )

    text = safe_text(
        path
    )


    if text is None:

        continue


    # Cheap file-level prefilter.
    if not (
        P95_REGEX.search(
            text
        )
        or
        "p95" in text.lower()
    ):

        continue


    file_sha = sha256_file(
        path
    )


    for target_id, metadata in PRIMARY_TARGETS.items():

        aliases = [
            target_id,
            *metadata[
                "aliases"
            ],
        ]


        if not alias_hit(
            text,
            aliases,
        ):

            continue


        record = {
            "repo_relative_path":
                rel,

            "sha256":
                file_sha,

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "aliases_matched":
                sorted(
                    set(
                        alias_hit(
                            text,
                            aliases,
                        )
                    )
                ),

            "structured_cost_evidence":
                [],

            "line_evidence":
                [],
        }


        # -------------------------------------------------------------
        # JSON
        # -------------------------------------------------------------

        if path.suffix.lower() == ".json":

            try:

                obj = json.loads(
                    text
                )

            except Exception:

                obj = None


            if obj is not None:

                for node_path, node in json_walk(
                    obj
                ):

                    if not isinstance(
                        node,
                        dict,
                    ):

                        continue


                    node_text = json.dumps(
                        node,
                        default=str,
                    )


                    if not alias_hit(
                        node_text,
                        aliases,
                    ):

                        continue


                    scalar = dict(
                        scalar_metric_pairs(
                            node
                        )
                    )


                    p95_pairs = [
                        (
                            key,
                            value,
                        )
                        for key, value in scalar.items()
                        if (
                            P95_REGEX.search(
                                key
                            )
                            or
                            (
                                "95"
                                in
                                normalize(
                                    key
                                )
                                and
                                LATENCY_REGEX.search(
                                    key
                                )
                            )
                        )
                    ]


                    if not p95_pairs:

                        continue


                    # Record candidate context. Do NOT silently choose it.
                    record[
                        "structured_cost_evidence"
                    ].append(
                        {
                            "json_path":
                                node_path,

                            "p95_pairs":
                                p95_pairs,

                            "context":
                                compact_dict(
                                    node,
                                    max_items=30,
                                ),
                        }
                    )


                    if len(
                        record[
                            "structured_cost_evidence"
                        ]
                    ) >= 15:

                        break


        # -------------------------------------------------------------
        # CSV
        # -------------------------------------------------------------

        elif path.suffix.lower() == ".csv":

            try:

                reader = csv.DictReader(
                    io.StringIO(
                        text
                    )
                )

                fieldnames = (
                    reader.fieldnames
                    or
                    []
                )


                p95_columns = [
                    column
                    for column in fieldnames
                    if (
                        P95_REGEX.search(
                            str(
                                column
                            )
                        )
                        or
                        (
                            "95"
                            in
                            normalize(
                                column
                            )
                            and
                            LATENCY_REGEX.search(
                                str(
                                    column
                                )
                            )
                        )
                    )
                ]


                if p95_columns:

                    for row_number, row in enumerate(
                        reader,
                        start=2,
                    ):

                        row_text = " | ".join(
                            f"{k}={v}"
                            for k, v in row.items()
                        )


                        if not alias_hit(
                            row_text,
                            aliases,
                        ):

                            continue


                        # Keep explicit batch/mode fields in context so
                        # CPU1/B1 can be resolved from output.
                        context = {
                            key:
                                value
                            for key, value in row.items()
                            if (
                                value not in {
                                    None,
                                    "",
                                }
                                and
                                (
                                    key in p95_columns
                                    or
                                    re.search(
                                        r"(?i)"
                                        r"(target|model|batch|cpu|mode|"
                                        r"affinity|thread|latency|status|"
                                        r"condition)",
                                        str(
                                            key
                                        ),
                                    )
                                )
                            )
                        }


                        record[
                            "structured_cost_evidence"
                        ].append(
                            {
                                "csv_row":
                                    row_number,

                                "p95_pairs": [
                                    (
                                        column,
                                        row.get(
                                            column
                                        ),
                                    )
                                    for column in p95_columns
                                ],

                                "context":
                                    context,
                            }
                        )


                        if len(
                            record[
                                "structured_cost_evidence"
                            ]
                        ) >= 15:

                            break

            except Exception:

                pass


        # -------------------------------------------------------------
        # Plain line evidence
        # -------------------------------------------------------------

        for line_number, line in enumerate(
            text.splitlines(),
            start=1,
        ):

            if not alias_hit(
                line,
                aliases,
            ):

                continue

            if not P95_REGEX.search(
                line
            ):

                continue


            record[
                "line_evidence"
            ].append(
                {
                    "line":
                        line_number,

                    "text":
                        line[
                            :1200
                        ],
                }
            )


            if len(
                record[
                    "line_evidence"
                ]
            ) >= 8:

                break


        if (
            record[
                "structured_cost_evidence"
            ]
            or
            record[
                "line_evidence"
            ]
        ):

            cost_candidates[
                target_id
            ].append(
                record
            )


for target_id in cost_candidates:

    cost_candidates[
        target_id
    ].sort(
        key=lambda row: (
            -len(
                row[
                    "structured_cost_evidence"
                ]
            ),
            row[
                "repo_relative_path"
            ],
        )
    )


for target_id, candidates in cost_candidates.items():

    print(
        "\n"
        +
        "-" * 124
    )

    print(
        target_id
    )

    print(
        "CPU1 B=1 cost-candidate artifacts:",
        len(
            candidates
        )
    )


    if not candidates:

        print(
            "  [NONE FOUND]"
        )

        continue


    for candidate_index, candidate in enumerate(
        candidates[
            :10
        ],
        start=1,
    ):

        print(
            f"\n  [{candidate_index}] "
            f"{candidate['repo_relative_path']}"
        )

        print(
            "      SHA256:",
            candidate[
                "sha256"
            ]
        )


        if candidate[
            "structured_cost_evidence"
        ]:

            print(
                "      STRUCTURED P95 EVIDENCE:"
            )

            for evidence in candidate[
                "structured_cost_evidence"
            ][
                :8
            ]:

                print(
                    "       ",
                    json.dumps(
                        evidence,
                        sort_keys=True,
                        default=str,
                    )[
                        :2500
                    ]
                )


        if candidate[
            "line_evidence"
        ]:

            print(
                "      LINE P95 EVIDENCE:"
            )

            for evidence in candidate[
                "line_evidence"
            ][
                :5
            ]:

                print(
                    f"        L{evidence['line']}: "
                    f"{evidence['text']}"
                )


# =============================================================================
# 9. OPERATIONAL / RESOURCE REFERENCE PRESENCE
# =============================================================================

banner(
    "STAGE26-6A :: REFERENCE-TARGET INVENTORY"
)


reference_presence = {}


all_stage26_warm_text = []


for rel in warm_paths:

    text = safe_text(
        REPO
        /
        rel
    )

    if text is not None:

        all_stage26_warm_text.append(
            (
                rel,
                text,
            )
        )


for target_id, metadata in REFERENCE_TARGETS.items():

    matches = []


    for rel, text in all_stage26_warm_text:

        hits = alias_hit(
            text,
            [
                target_id,
                *metadata[
                    "aliases"
                ],
            ],
        )


        if hits:

            matches.append(
                {
                    "repo_relative_path":
                        rel,

                    "sha256":
                        sha256_file(
                            REPO
                            /
                            rel
                        ),

                    "aliases_matched":
                        sorted(
                            set(
                                hits
                            )
                        ),
                }
            )


    reference_presence[
        target_id
    ] = {
        "role":
            metadata[
                "role"
            ],

        "artifact_matches":
            matches,
    }


    print(
        "\n",
        target_id,
        sep="",
    )

    print(
        "  role:",
        metadata[
            "role"
        ]
    )

    print(
        "  Stage26-2 artifact matches:",
        len(
            matches
        )
    )


    for match in matches[
        :10
    ]:

        print(
            "   ",
            match[
                "repo_relative_path"
            ]
        )


# =============================================================================
# 10. RESOLUTION STATUS — DO NOT SELECT YET
# =============================================================================

banner(
    "STAGE26-6A :: SOURCE-RESOLUTION STATUS"
)


resolution_status = {}


for target_id in PRIMARY_TARGETS:

    metric_count = len(
        predictive_candidates[
            target_id
        ]
    )

    cost_count = len(
        cost_candidates[
            target_id
        ]
    )


    metric_structured_count = sum(
        1
        for candidate in predictive_candidates[
            target_id
        ]
        if candidate[
            "structured_metric_evidence"
        ]
    )

    cost_structured_count = sum(
        1
        for candidate in cost_candidates[
            target_id
        ]
        if candidate[
            "structured_cost_evidence"
        ]
    )


    resolution_status[
        target_id
    ] = {
        "group":
            PRIMARY_TARGETS[
                target_id
            ][
                "group"
            ],

        "predictive_candidate_file_count":
            metric_count,

        "predictive_structured_candidate_file_count":
            metric_structured_count,

        "cost_candidate_file_count":
            cost_count,

        "cost_structured_candidate_file_count":
            cost_structured_count,

        "source_selected":
            False,

        "pareto_computed":
            False,
    }


    print(
        f"{target_id:42s} | "
        f"PR files={metric_count:2d} "
        f"(structured={metric_structured_count:2d}) | "
        f"cost files={cost_count:2d} "
        f"(structured={cost_structured_count:2d})"
    )


# =============================================================================
# 11. WRITE TRANSIENT INVENTORY
# =============================================================================

banner(
    "STAGE26-6A :: WRITE TRANSIENT SOURCE INVENTORY"
)


RUNTIME_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


inventory = {
    "schema":
        "stage26_6a_pareto_source_resolution_inventory_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "pareto_protocol":
        pareto,

    "primary_targets":
        PRIMARY_TARGETS,

    "reference_targets":
        REFERENCE_TARGETS,

    "predictive_metric_candidates":
        predictive_candidates,

    "cpu1_batch1_p95_cost_candidates":
        cost_candidates,

    "reference_target_presence":
        reference_presence,

    "resolution_status":
        resolution_status,

    "scientific_state": {
        "predictive_metric_recomputed":
            False,

        "holdout_opened":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "pareto_dominance_computed":
            False,

        "pareto_sources_selected":
            False,

        "pcap_accessed":
            False,

        "release_corpus_accessed":
            False,

        "gpu_used":
            False,

        "git_modified":
            False,
    },
}


with RUNTIME_OUT.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        inventory,
        f,
        indent=2,
        sort_keys=True,
    )

    f.write(
        "\n"
    )


inventory_sha = sha256_file(
    RUNTIME_OUT
)


print(
    "Inventory:"
)

print(
    " ",
    RUNTIME_OUT
)

print(
    "SHA256:"
)

print(
    " ",
    inventory_sha
)


# =============================================================================
# 12. FINAL GIT / SCIENTIFIC AUDIT
# =============================================================================

banner(
    "STAGE26-6A SOURCE-RESOLUTION INVENTORY COMPLETE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:

    raise RuntimeError(
        "Git HEAD changed during read-only inventory."
    )


if final_remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed during source resolution."
    )


if final_status:

    raise RuntimeError(
        "Read-only Stage26-6A unexpectedly modified Git."
    )


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  committed predictive artifacts inspected : YES"
)

print(
    "  Stage26-2 cost artifacts inspected        : YES"
)

print(
    "  PR-AUC recomputed                         : NO"
)

print(
    "  holdout reopened                          : NO"
)

print(
    "  model loaded                              : NO"
)

print(
    "  inference performed                       : NO"
)

print(
    "  timing performed                          : NO"
)

print(
    "  Pareto source selected                    : NO"
)

print(
    "  Pareto dominance computed                 : NO"
)

print(
    "  cross-group comparison                    : NO"
)

print(
    "  GPU                                       : NO"
)

print(
    "  Git modified                              : NO"
)


print(
    "\nNEXT AFTER REVIEWING THIS OUTPUT:"
)

print(
    "  Resolve exactly ONE frozen predictive metric source/value"
)

print(
    "  and exactly ONE CPU1 B=1 p95 inference-cost source/value"
)

print(
    "  for each eligible primary target."
)

print(
    "  Then freeze those six Pareto points BEFORE computing dominance."
)


STAGE26-6A :: DURABLE GIT GATE
Expected parent: 92495eac2c9202973ff4d889e3c12669c06e9ef1
Local HEAD     : 92495eac2c9202973ff4d889e3c12669c06e9ef1
origin/main    : 92495eac2c9202973ff4d889e3c12669c06e9ef1
Repo clean     : True

STAGE26-6A :: FROZEN PARETO PROTOCOL
Protocol SHA256:
 expected: d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
 actual  : d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625

GROUP A:
{
  "eligible_primary_members": [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING"
  ],
  "primary_cost_axis": "CPU_1_PHYSICAL_CORE batch1 p95 inference latency",
  "primary_discrimination_axis": "Same-population frozen PR-AUC"
}

GROUP B:
{
  "claim_boundary": "DESCRIPTIVE_NON_CONFIRMATORY",
  "eligible_primary_members": [
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1"
  ],
  "primary_cost_axis": "CPU_1_PHYSICAL_CORE batch1 p95 inference latency",
  "primar

In [12]:
# =============================================================================
# STAGE26-6B
# EXACT PARETO-POINT SOURCE RESOLUTION — NO DOMINANCE
#
# SCIENTIFIC PARENT:
#   92495eac2c9202973ff4d889e3c12669c06e9ef1
#
# PURPOSE
# -------
# Resolve exactly six primary Stage26 Pareto points:
#
# GROUP A — exact duplicate-safe VALIDATION population
#   XGBoost
#   LightGBM
#   CatBoost
#   FT 5-checkpoint soft-voting ensemble
#
# GROUP B — exact Friday locked reuse population
#   CNN
#   ViT
#
# Each point contains:
#   - frozen PR-AUC
#   - exact predictive source artifact / field
#   - CPU1 batch-1 p95 inference latency
#   - exact Stage26-2 source row / condition
#
# CRITICAL:
#   NO dominance calculation
#   NO bootstrap calculation
#   NO holdout reopening
#   NO metric recomputation
#   NO model loading
#   NO inference
#   NO timing
#   NO PCAP
#   NO Release corpus
#   NO GPU
#   NO Git write
#
# Output is TRANSIENT only:
#   /kaggle/working/stage26_deployment_profiling/pareto/
#       stage26_6b_resolved_pareto_points.json
# =============================================================================

from __future__ import annotations

import csv
import json
import math
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "92495eac2c9202973ff4d889e3c12669c06e9ef1"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

PARETO_RUNTIME = (
    STAGE26_ROOT
    / "pareto"
)

SOURCE_6A = (
    PARETO_RUNTIME
    / "stage26_6a_source_resolution_inventory.json"
)

EXPECTED_6A_SHA256 = (
    "bdf56d8e972206b8186fcc1580c7b35ea60cae58c33fe4702fd1cc56e7ff12cb"
)

OUT = (
    PARETO_RUNTIME
    / "stage26_6b_resolved_pareto_points.json"
)


# -----------------------------------------------------------------------------
# Stage26-0 pre-measurement locks
# -----------------------------------------------------------------------------

LOCK0 = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
)

MEASUREMENT_PROTOCOL = (
    LOCK0
    / "measurement_protocol.json"
)

EXPECTED_MEASUREMENT_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

PACKAGE_0C_MANIFEST = (
    LOCK0
    / "stage26_0c_package_manifest.json"
)

RESOLVED_ARTIFACTS = (
    LOCK0
    / "stage26_0b_resolved_artifacts.csv"
)

EXPECTED_RESOLVED_ARTIFACTS_SHA256 = (
    "bb73dd1c63ab78942cea9eba9c1d1bc9d278cb3ae4963e8c7410b37ef8d270ad"
)

COMPARABILITY_GROUPS = (
    LOCK0
    / "stage26_0b_comparability_groups.json"
)

EXPECTED_COMPARABILITY_SHA256 = (
    "44d54cfa97360a094a6c86cb3e5d447875ebe79f0a0e5232b96b174abcf459ca"
)

CANDIDATE_REGISTRY = (
    LOCK0
    / "stage26_0b_candidate_registry.json"
)

EXPECTED_CANDIDATE_REGISTRY_SHA256 = (
    "610172f5324d81dccf40d536c9c4a7a22953404dfb67a4c5bf4d197610ca608a"
)


# -----------------------------------------------------------------------------
# Historical frozen predictive artifacts
# -----------------------------------------------------------------------------

FT_GAP = (
    REPO
    / "results"
    / "stage15_transformer_checkpoint"
    / "stage15_6a_ensemble_validation_holdout_gap.json"
)

EXPECTED_FT_GAP_SHA256 = (
    "d95ae4343f21fe6db241b6e03425120dbff568efaeb3f6e47aa21fc8da6196a7"
)

CNN_VIT = (
    REPO
    / "results"
    / "stage21_architecture"
    / "stage21_5_cnn_vit_descriptive_comparison.json"
)

EXPECTED_CNN_VIT_SHA256 = (
    "142bef3193ed1e9acaaee60bfcf4c0ce49516c58a2487d7bb6d5519a1f2a7885"
)


# -----------------------------------------------------------------------------
# Stage26-2 cost artifacts
# -----------------------------------------------------------------------------

WARM_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_2_cpu_warm_inference"
)

WARM_SUMMARY = (
    WARM_DIR
    / "stage26_2_warm_summary.csv"
)

EXPECTED_WARM_SUMMARY_SHA256 = (
    "75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232"
)

CONDITION_STATUS = (
    WARM_DIR
    / "stage26_2_condition_status.csv"
)

EXPECTED_CONDITION_STATUS_SHA256 = (
    "df146563826992cb57702e589e4cf5d875ecfdbbf79533a731405e8eb738e7af"
)


# =============================================================================
# 1. FROZEN TARGET UNIVERSE
# =============================================================================

GROUP_A = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
]

GROUP_B = [
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
]

ALL_PRIMARY = (
    GROUP_A
    +
    GROUP_B
)


EXPECTED_PR_AUC = {
    "STAGE16_XGBOOST_TUNED":
        0.9453847557659393,

    "STAGE16_LIGHTGBM_TUNED":
        0.9466063189077636,

    "STAGE16_CATBOOST_TUNED":
        0.9430447292850749,

    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING":
        0.9295715928199866,

    "STAGE20_MASKED_CNN_V1":
        0.48945269459245255,

    "STAGE21_MASKED_VIT_V1":
        0.606536911289453,
}


# =============================================================================
# 2. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        +
        "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            p.stdout
        )

    return (
        p.stdout
        or
        ""
    ).strip()


def git(*args):

    return run(
        [
            "git",
            *args,
        ]
    )


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8
                *
                1024
                *
                1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def rel(path):

    return str(
        Path(path).relative_to(
            REPO
        )
    )


def load_csv_rows(path):

    with Path(path).open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def assert_float_exact(
    actual,
    expected,
    label,
):

    actual = float(
        actual
    )

    expected = float(
        expected
    )

    if actual != expected:

        raise RuntimeError(
            f"{label} mismatch:\n"
            f"expected={expected!r}\n"
            f"actual  ={actual!r}"
        )

    return actual


# =============================================================================
# 3. DURABLE GIT GATE
# =============================================================================

banner(
    "STAGE26-6B :: DURABLE GIT GATE"
)

git(
    "fetch",
    "origin",
    "main",
)

head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Unexpected Stage26-6B scientific parent."
    )

if remote != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main changed before Stage26-6B."
    )

if status:
    raise RuntimeError(
        "Repository must be clean."
    )


# =============================================================================
# 4. STAGE26-6A CONTINUITY GATE
# =============================================================================

banner(
    "STAGE26-6B :: STAGE26-6A CONTINUITY"
)

if not SOURCE_6A.is_file():

    raise RuntimeError(
        "Stage26-6A transient inventory is missing."
    )


actual_6a_sha = sha256_file(
    SOURCE_6A
)


print(
    "Expected 6A SHA256:",
    EXPECTED_6A_SHA256
)

print(
    "Actual   6A SHA256:",
    actual_6a_sha
)


if actual_6a_sha != EXPECTED_6A_SHA256:

    raise RuntimeError(
        "Stage26-6A inventory identity mismatch."
    )


inventory_6a = json.loads(
    SOURCE_6A.read_text(
        encoding="utf-8"
    )
)


for target in ALL_PRIMARY:

    state = inventory_6a[
        "resolution_status"
    ][
        target
    ]

    if state[
        "source_selected"
    ] is not False:

        raise RuntimeError(
            f"6A unexpectedly selected source for {target}."
        )

    if state[
        "pareto_computed"
    ] is not False:

        raise RuntimeError(
            f"6A unexpectedly computed Pareto for {target}."
        )


print(
    "6A source selection : NONE"
)

print(
    "6A Pareto computed  : NO"
)


# =============================================================================
# 5. FROZEN SOURCE HASH GATE
# =============================================================================

banner(
    "STAGE26-6B :: FROZEN SOURCE IDENTITY"
)


hash_expectations = [
    (
        MEASUREMENT_PROTOCOL,
        EXPECTED_MEASUREMENT_PROTOCOL_SHA256,
        "measurement protocol",
    ),
    (
        RESOLVED_ARTIFACTS,
        EXPECTED_RESOLVED_ARTIFACTS_SHA256,
        "Stage26-0B resolved artifacts",
    ),
    (
        COMPARABILITY_GROUPS,
        EXPECTED_COMPARABILITY_SHA256,
        "Stage26-0B comparability groups",
    ),
    (
        CANDIDATE_REGISTRY,
        EXPECTED_CANDIDATE_REGISTRY_SHA256,
        "Stage26-0B candidate registry",
    ),
    (
        FT_GAP,
        EXPECTED_FT_GAP_SHA256,
        "Stage15 FT validation/holdout gap",
    ),
    (
        CNN_VIT,
        EXPECTED_CNN_VIT_SHA256,
        "Stage21 CNN/ViT comparison",
    ),
    (
        WARM_SUMMARY,
        EXPECTED_WARM_SUMMARY_SHA256,
        "Stage26-2 warm summary",
    ),
    (
        CONDITION_STATUS,
        EXPECTED_CONDITION_STATUS_SHA256,
        "Stage26-2 condition status",
    ),
]


for path, expected_sha, label in hash_expectations:

    actual_sha = sha256_file(
        path
    )

    print(
        f"{label:39s} "
        f"{'PASS' if actual_sha == expected_sha else 'FAIL'} "
        f"{actual_sha}"
    )

    if actual_sha != expected_sha:

        raise RuntimeError(
            f"{label} SHA256 mismatch."
        )


# =============================================================================
# 6. VERIFY STAGE26-0C MANIFEST ITSELF AGREES
# =============================================================================

banner(
    "STAGE26-6B :: STAGE26-0C MANIFEST CROSS-CHECK"
)


manifest_0c = json.loads(
    PACKAGE_0C_MANIFEST.read_text(
        encoding="utf-8"
    )
)


manifest_files = {
    item[
        "path"
    ]:
        item
    for item in manifest_0c[
        "files"
    ]
}


manifest_required = {
    rel(
        MEASUREMENT_PROTOCOL
    ):
        EXPECTED_MEASUREMENT_PROTOCOL_SHA256,

    rel(
        RESOLVED_ARTIFACTS
    ):
        EXPECTED_RESOLVED_ARTIFACTS_SHA256,

    rel(
        COMPARABILITY_GROUPS
    ):
        EXPECTED_COMPARABILITY_SHA256,

    rel(
        CANDIDATE_REGISTRY
    ):
        EXPECTED_CANDIDATE_REGISTRY_SHA256,
}


for path_string, expected_sha in manifest_required.items():

    if path_string not in manifest_files:

        raise RuntimeError(
            f"Stage26-0C manifest missing:\n{path_string}"
        )

    manifest_sha = manifest_files[
        path_string
    ][
        "sha256"
    ]

    if manifest_sha != expected_sha:

        raise RuntimeError(
            f"Stage26-0C manifest SHA disagreement:\n"
            f"{path_string}"
        )

    print(
        "PASS",
        expected_sha,
        path_string,
    )


print(
    "\nStage26-0C manifest SHA256:",
    sha256_file(
        PACKAGE_0C_MANIFEST
    )
)


# =============================================================================
# 7. COMPARABILITY / CANDIDATE-UNIVERSE GATE
# =============================================================================

banner(
    "STAGE26-6B :: COMPARABILITY GATE"
)


comparability = json.loads(
    COMPARABILITY_GROUPS.read_text(
        encoding="utf-8"
    )
)

registry = json.loads(
    CANDIDATE_REGISTRY.read_text(
        encoding="utf-8"
    )
)


groups = comparability[
    "groups"
]


if groups[
    "GROUP_A_DUPSAFE70"
][
    "members"
] != GROUP_A:

    raise RuntimeError(
        "Frozen Group-A member universe mismatch."
    )


if groups[
    "GROUP_B_PACKET_IMAGE"
][
    "members"
] != GROUP_B:

    raise RuntimeError(
        "Frozen Group-B member universe mismatch."
    )


group_a_semantics = groups[
    "GROUP_A_DUPSAFE70"
][
    "population_semantics"
]

group_b_semantics = groups[
    "GROUP_B_PACKET_IMAGE"
][
    "population_semantics"
]


if "Stage15" not in group_a_semantics:

    raise RuntimeError(
        "Group-A Stage15 population inheritance missing."
    )


if "Stage16" not in group_a_semantics:

    raise RuntimeError(
        "Group-A Stage16 population inheritance missing."
    )


if groups[
    "GROUP_B_PACKET_IMAGE"
][
    "friday_rows"
] != 12088:

    raise RuntimeError(
        "Friday row-count mismatch."
    )


if groups[
    "GROUP_B_PACKET_IMAGE"
][
    "friday_benign"
] != 6486:

    raise RuntimeError(
        "Friday benign-count mismatch."
    )


if groups[
    "GROUP_B_PACKET_IMAGE"
][
    "friday_attack"
] != 5602:

    raise RuntimeError(
        "Friday attack-count mismatch."
    )


if (
    "NO CROSS-GROUP PARETO FRONTIER"
    not in
    comparability[
        "global_rule"
    ]
):

    raise RuntimeError(
        "Cross-group Pareto prohibition missing."
    )


if (
    "DESCRIPTIVE"
    not in
    groups[
        "GROUP_B_PACKET_IMAGE"
    ][
        "claim_boundary"
    ]
):

    raise RuntimeError(
        "Group-B descriptive claim boundary missing."
    )


registry_primary = [
    item[
        "model_id"
    ]
    for item in registry[
        "primary_model_universe"
    ]
]


if registry_primary != ALL_PRIMARY:

    raise RuntimeError(
        "Stage26 primary model universe mismatch."
    )


references = registry[
    "operational_references"
]


if references[
    "ENS_LGBM_XGB_EQUAL"
][
    "count_as_architecture_family"
] is not False:

    raise RuntimeError(
        "Operational ensemble architecture-count rule changed."
    )


if references[
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE"
][
    "eligible_for_predictive_pareto"
] is not False:

    raise RuntimeError(
        "FT single resource-reference exclusion changed."
    )


print(
    "Group A population:",
    group_a_semantics
)

print(
    "Group B population:",
    group_b_semantics
)

print(
    "Group B Friday   :",
    "12,088 flows = 6,486 benign + 5,602 attack"
)

print(
    "Cross-group Pareto:",
    "FORBIDDEN"
)

print(
    "FT single resource reference:",
    "EXCLUDED"
)


# =============================================================================
# 8. RESOLVE GROUP-A CLASSICAL PR-AUC
# =============================================================================

banner(
    "STAGE26-6B :: GROUP-A CLASSICAL PREDICTIVE POINTS"
)


resolved_rows = load_csv_rows(
    RESOLVED_ARTIFACTS
)


classical_targets = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
]


pr_sources = {}


for target in classical_targets:

    matches = [
        (
            row_number,
            row,
        )
        for row_number, row in enumerate(
            resolved_rows,
            start=2,
        )
        if row.get(
            "artifact_id"
        ) == target
    ]


    if len(
        matches
    ) != 1:

        raise RuntimeError(
            f"{target}: expected exactly one Stage26-0B row; "
            f"found {len(matches)}."
        )


    row_number, row = matches[
        0
    ]


    if row[
        "comparison_group"
    ] != "GROUP_A_DUPSAFE70":

        raise RuntimeError(
            f"{target}: wrong comparison group."
        )


    if row[
        "role"
    ] != "PRIMARY_ARCHITECTURE_PROFILE":

        raise RuntimeError(
            f"{target}: wrong Stage26 role."
        )


    note = row[
        "notes"
    ].lower()


    if "duplicate-safe" not in note:

        raise RuntimeError(
            f"{target}: duplicate-safe provenance not explicit."
        )


    if "validation only" not in note:

        raise RuntimeError(
            f"{target}: validation-only selection provenance missing."
        )


    pr_auc = assert_float_exact(
        row[
            "validation_pr_auc"
        ],
        EXPECTED_PR_AUC[
            target
        ],
        f"{target} validation PR-AUC",
    )


    pr_sources[
        target
    ] = {
        "value":
            pr_auc,

        "population":
            "STAGE15_STAGE16_EXACT_DUPSAFE_VALIDATION",

        "source_path":
            rel(
                RESOLVED_ARTIFACTS
            ),

        "source_sha256":
            EXPECTED_RESOLVED_ARTIFACTS_SHA256,

        "source_row":
            row_number,

        "source_field":
            "validation_pr_auc",

        "configuration_id":
            row[
                "winning_configuration_id"
            ],

        "selected_threshold":
            float(
                row[
                    "selected_threshold"
                ]
            ),
    }


    print(
        f"{target:42s} "
        f"PR-AUC={pr_auc:.15f} "
        f"config={row['winning_configuration_id']}"
    )


# =============================================================================
# 9. RESOLVE GROUP-A FT ENSEMBLE PR-AUC
# =============================================================================

banner(
    "STAGE26-6B :: GROUP-A FT-ENSEMBLE PREDICTIVE POINT"
)


ft = json.loads(
    FT_GAP.read_text(
        encoding="utf-8"
    )
)


if ft[
    "model"
] != "five_checkpoint_soft_voting_ensemble":

    raise RuntimeError(
        "Unexpected Stage15 FT predictor."
    )


if ft[
    "ensemble_method"
] != "unweighted arithmetic mean probabilities":

    raise RuntimeError(
        "Unexpected FT ensemble method."
    )


ft_pr = assert_float_exact(
    ft[
        "validation_pr_auc"
    ],
    EXPECTED_PR_AUC[
        "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING"
    ],
    "FT ensemble validation PR-AUC",
)


pr_sources[
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING"
] = {
    "value":
        ft_pr,

    "population":
        "STAGE15_STAGE16_EXACT_DUPSAFE_VALIDATION",

    "source_path":
        rel(
            FT_GAP
        ),

    "source_sha256":
        EXPECTED_FT_GAP_SHA256,

    "source_field":
        "validation_pr_auc",

    "explicitly_not_used":
        {
            "field":
                "holdout_pr_auc",

            "value":
                float(
                    ft[
                        "holdout_pr_auc"
                    ]
                ),

            "reason":
                (
                    "Group-A primary classical architecture points "
                    "are frozen on the shared duplicate-safe "
                    "validation population."
                ),
        },

    "ensemble_method":
        ft[
            "ensemble_method"
        ],

    "selected_threshold":
        float(
            ft[
                "threshold"
        ]
    ),
}


print(
    "FT validation PR-AUC:",
    f"{ft_pr:.15f}"
)

print(
    "FT holdout PR-AUC   :",
    f"{float(ft['holdout_pr_auc']):.15f}",
    "(NOT USED)"
)


# =============================================================================
# 10. RESOLVE GROUP-B SAME-FRIDAY PR-AUC
# =============================================================================

banner(
    "STAGE26-6B :: GROUP-B SAME-FRIDAY PREDICTIVE POINTS"
)


cnn_vit = json.loads(
    CNN_VIT.read_text(
        encoding="utf-8"
    )
)


if cnn_vit[
    "status"
] != "PREREGISTERED_DESCRIPTIVE_CNN_VIT_COMPARISON_COMPLETE":

    raise RuntimeError(
        "Stage21 comparison is not in expected final state."
    )


if cnn_vit[
    "role"
] != "LOCKED_FRIDAY_REUSE_BENCHMARK_NON_CONFIRMATORY":

    raise RuntimeError(
        "Unexpected Stage21 Friday role."
    )


population = cnn_vit[
    "population"
]


if population != {
    "attack": 5602,
    "benign": 6486,
    "flows": 12088,
    "pairing": "EXACT_SAME_FRIDAY_COMPACT_CORPUS_EXPORT_ORDER",
}:

    raise RuntimeError(
        "Stage21 Friday population identity mismatch."
    )


co_primary = cnn_vit[
    "co_primary_descriptive"
]


group_b_fields = {
    "STAGE20_MASKED_CNN_V1":
        "CNN_PR_AUC",

    "STAGE21_MASKED_VIT_V1":
        "ViT_PR_AUC",
}


for target, field in group_b_fields.items():

    pr_auc = assert_float_exact(
        co_primary[
            field
        ],
        EXPECTED_PR_AUC[
            target
        ],
        f"{target} Friday PR-AUC",
    )


    pr_sources[
        target
    ] = {
        "value":
            pr_auc,

        "population":
            "EXACT_SAME_FRIDAY_COMPACT_CORPUS_EXPORT_ORDER",

        "population_rows":
            12088,

        "population_benign":
            6486,

        "population_attack":
            5602,

        "claim_boundary":
            "DESCRIPTIVE_NON_CONFIRMATORY",

        "source_path":
            rel(
                CNN_VIT
            ),

        "source_sha256":
            EXPECTED_CNN_VIT_SHA256,

        "source_field":
            (
                "co_primary_descriptive."
                +
                field
            ),
    }


    print(
        f"{target:42s} "
        f"PR-AUC={pr_auc:.15f}"
    )


# =============================================================================
# 11. RESOLVE CPU1 B=1 P95 COST POINTS
# =============================================================================

banner(
    "STAGE26-6B :: CPU1 B=1 P95 INFERENCE COST"
)


warm_rows = load_csv_rows(
    WARM_SUMMARY
)

status_rows = load_csv_rows(
    CONDITION_STATUS
)


cost_sources = {}


for target in ALL_PRIMARY:

    summary_matches = [
        (
            row_number,
            row,
        )
        for row_number, row in enumerate(
            warm_rows,
            start=2,
        )
        if (
            row.get(
                "target_id"
            )
            ==
            target
            and
            row.get(
                "hardware_mode"
            )
            ==
            "CPU_1_PHYSICAL_CORE"
            and
            int(
                row.get(
                    "batch_size"
                )
            )
            ==
            1
            and
            int(
                row.get(
                    "thread_count"
                )
            )
            ==
            1
        )
    ]


    if len(
        summary_matches
    ) != 1:

        raise RuntimeError(
            f"{target}: expected exactly one CPU1/B1 "
            f"warm-summary row; found {len(summary_matches)}."
        )


    summary_row_number, summary_row = summary_matches[
        0
    ]


    condition_id = summary_row[
        "condition_id"
    ]


    if int(
        summary_row[
            "n"
        ]
    ) != 200:

        raise RuntimeError(
            f"{target}: CPU1/B1 expected n=200."
        )


    status_matches = [
        (
            row_number,
            row,
        )
        for row_number, row in enumerate(
            status_rows,
            start=2,
        )
        if (
            row.get(
                "target_id"
            )
            ==
            target
            and
            row.get(
                "condition_id"
            )
            ==
            condition_id
            and
            row.get(
                "hardware_mode"
            )
            ==
            "CPU_1_PHYSICAL_CORE"
            and
            int(
                row.get(
                    "batch_size"
                )
            )
            ==
            1
            and
            int(
                row.get(
                    "thread_count"
                )
            )
            ==
            1
        )
    ]


    if len(
        status_matches
    ) != 1:

        raise RuntimeError(
            f"{target}: expected exactly one matching "
            f"condition-status row; found {len(status_matches)}."
        )


    status_row_number, status_row = status_matches[
        0
    ]


    if status_row[
        "status"
    ] != "PASS":

        raise RuntimeError(
            f"{target}: CPU1/B1 condition is not PASS."
        )


    summary_p95 = float(
        summary_row[
            "p95_batch_latency_ms"
        ]
    )

    status_p95 = float(
        status_row[
            "p95_batch_latency_ms"
        ]
    )


    if summary_p95 != status_p95:

        raise RuntimeError(
            f"{target}: Stage26-2 p95 disagreement:\n"
            f"summary={summary_p95!r}\n"
            f"status ={status_p95!r}"
        )


    cost_sources[
        target
    ] = {
        "value_ms":
            summary_p95,

        "metric":
            "CPU_1_PHYSICAL_CORE_BATCH1_P95_INFERENCE_LATENCY_MS",

        "hardware_mode":
            "CPU_1_PHYSICAL_CORE",

        "affinity":
            [0],

        "thread_count":
            1,

        "batch_size":
            1,

        "timed_runs":
            200,

        "condition_id":
            condition_id,

        "summary_source_path":
            rel(
                WARM_SUMMARY
            ),

        "summary_source_sha256":
            EXPECTED_WARM_SUMMARY_SHA256,

        "summary_source_row":
            summary_row_number,

        "summary_source_field":
            "p95_batch_latency_ms",

        "status_crosscheck_path":
            rel(
                CONDITION_STATUS
            ),

        "status_crosscheck_sha256":
            EXPECTED_CONDITION_STATUS_SHA256,

        "status_crosscheck_row":
            status_row_number,

        "status":
            status_row[
                "status"
            ],
    }


    print(
        f"{target:42s} "
        f"{condition_id:11s} "
        f"p95={summary_p95:.12f} ms"
    )


# =============================================================================
# 12. BUILD EXACT SIX POINT RECORDS
# =============================================================================

banner(
    "STAGE26-6B :: RESOLVED SIX PRIMARY POINTS"
)


points = []


for target in ALL_PRIMARY:

    group = (
        "GROUP_A_DUPSAFE70"
        if target in GROUP_A
        else
        "GROUP_B_PACKET_IMAGE"
    )


    point = {
        "target_id":
            target,

        "comparison_group":
            group,

        "pr_auc":
            pr_sources[
                target
            ][
                "value"
            ],

        "p95_cpu1_batch1_inference_latency_ms":
            cost_sources[
                target
            ][
                "value_ms"
            ],

        "predictive_source":
            pr_sources[
                target
            ],

        "cost_source":
            cost_sources[
                target
            ],
    }


    points.append(
        point
    )


print(
    f"{'TARGET':42s} "
    f"{'GROUP':22s} "
    f"{'PR-AUC':>18s} "
    f"{'CPU1 B1 p95 ms':>18s}"
)

print(
    "-" * 104
)


for point in points:

    print(
        f"{point['target_id']:42s} "
        f"{point['comparison_group']:22s} "
        f"{point['pr_auc']:18.15f} "
        f"{point['p95_cpu1_batch1_inference_latency_ms']:18.12f}"
    )


# =============================================================================
# 13. POINT-SET FINGERPRINT
# =============================================================================

point_core = [
    {
        "target_id":
            item[
                "target_id"
            ],

        "comparison_group":
            item[
                "comparison_group"
            ],

        "pr_auc":
            item[
                "pr_auc"
            ],

        "p95_cpu1_batch1_inference_latency_ms":
            item[
                "p95_cpu1_batch1_inference_latency_ms"
            ],
    }
    for item in points
]


point_core_bytes = json.dumps(
    point_core,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
    allow_nan=False,
).encode(
    "utf-8"
)


point_set_sha256 = hashlib.sha256(
    point_core_bytes
).hexdigest()


print(
    "\nSix-point core SHA256:"
)

print(
    " ",
    point_set_sha256
)


# =============================================================================
# 14. WRITE TRANSIENT RESOLUTION RECEIPT
# =============================================================================

banner(
    "STAGE26-6B :: WRITE TRANSIENT RESOLUTION RECEIPT"
)


PARETO_RUNTIME.mkdir(
    parents=True,
    exist_ok=True,
)


payload = {
    "schema":
        "stage26_6b_resolved_pareto_points_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "stage26_6a_inventory": {
        "path":
            str(
                SOURCE_6A
            ),

        "sha256":
            EXPECTED_6A_SHA256,
    },

    "source_identities": {
        "measurement_protocol": {
            "path":
                rel(
                    MEASUREMENT_PROTOCOL
                ),

            "sha256":
                EXPECTED_MEASUREMENT_PROTOCOL_SHA256,
        },

        "stage26_0b_resolved_artifacts": {
            "path":
                rel(
                    RESOLVED_ARTIFACTS
                ),

            "sha256":
                EXPECTED_RESOLVED_ARTIFACTS_SHA256,
        },

        "stage26_0b_comparability_groups": {
            "path":
                rel(
                    COMPARABILITY_GROUPS
                ),

            "sha256":
                EXPECTED_COMPARABILITY_SHA256,
        },

        "stage26_0b_candidate_registry": {
            "path":
                rel(
                    CANDIDATE_REGISTRY
                ),

            "sha256":
                EXPECTED_CANDIDATE_REGISTRY_SHA256,
        },

        "ft_validation_source": {
            "path":
                rel(
                    FT_GAP
                ),

            "sha256":
                EXPECTED_FT_GAP_SHA256,
        },

        "cnn_vit_friday_source": {
            "path":
                rel(
                    CNN_VIT
                ),

            "sha256":
                EXPECTED_CNN_VIT_SHA256,
        },

        "warm_summary": {
            "path":
                rel(
                    WARM_SUMMARY
                ),

            "sha256":
                EXPECTED_WARM_SUMMARY_SHA256,
        },

        "condition_status": {
            "path":
                rel(
                    CONDITION_STATUS
                ),

            "sha256":
                EXPECTED_CONDITION_STATUS_SHA256,
        },
    },

    "comparability": {
        "GROUP_A_DUPSAFE70":
            groups[
                "GROUP_A_DUPSAFE70"
            ],

        "GROUP_B_PACKET_IMAGE":
            groups[
                "GROUP_B_PACKET_IMAGE"
            ],

        "global_rule":
            comparability[
                "global_rule"
            ],
    },

    "point_set_sha256":
        point_set_sha256,

    "points":
        points,

    "scientific_state": {
        "six_primary_points_resolved":
            True,

        "predictive_metric_recomputed":
            False,

        "holdout_reopened":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "bootstrap_computed":
            False,

        "dominance_computed":
            False,

        "pareto_frontier_computed":
            False,

        "cross_group_comparison":
            False,

        "operational_reference_added_as_primary":
            False,

        "ft_single_resource_reference_added_to_pareto":
            False,

        "pcap_accessed":
            False,

        "release_corpus_accessed":
            False,

        "gpu_used":
            False,

        "git_modified":
            False,
    },

    "next":
        (
            "Git-freeze these six resolved point identities "
            "before any dominance computation."
        ),
}


OUT.write_text(
    json.dumps(
        payload,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    )
    +
    "\n",
    encoding="utf-8",
)


out_sha = sha256_file(
    OUT
)


print(
    "Output:"
)

print(
    " ",
    OUT
)

print(
    "SHA256:"
)

print(
    " ",
    out_sha
)


# =============================================================================
# 15. FINAL GIT / SCIENTIFIC AUDIT
# =============================================================================

banner(
    "STAGE26-6B EXACT POINT RESOLUTION COMPLETE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:

    raise RuntimeError(
        "Git HEAD changed during Stage26-6B."
    )


if final_remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed during Stage26-6B."
    )


if final_status:

    raise RuntimeError(
        "Stage26-6B unexpectedly modified the repository."
    )


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  six primary point sources resolved : YES"
)

print(
    "  same-population Group A            : YES"
)

print(
    "  same-Friday Group B                : YES"
)

print(
    "  Group B claim boundary             : DESCRIPTIVE / NON-CONFIRMATORY"
)

print(
    "  CPU1 B1 p95 independently checked  : YES"
)

print(
    "  PR-AUC recomputed                   : NO"
)

print(
    "  holdout reopened                    : NO"
)

print(
    "  new inference                       : NO"
)

print(
    "  new timing                          : NO"
)

print(
    "  bootstrap computed                  : NO"
)

print(
    "  Pareto dominance computed           : NO"
)

print(
    "  cross-group Pareto                  : NO"
)

print(
    "  GPU                                 : NO"
)

print(
    "  Git modified                        : NO"
)


print(
    "\nNEXT ONLY AFTER REVIEW:"
)

print(
    "  Freeze this exact six-point source map in Git."
)

print(
    "  Do NOT compute dominance before that freeze is remotely anchored."
)


STAGE26-6B :: DURABLE GIT GATE
Expected parent: 92495eac2c9202973ff4d889e3c12669c06e9ef1
Local HEAD     : 92495eac2c9202973ff4d889e3c12669c06e9ef1
origin/main    : 92495eac2c9202973ff4d889e3c12669c06e9ef1
Repo clean     : True

STAGE26-6B :: STAGE26-6A CONTINUITY
Expected 6A SHA256: bdf56d8e972206b8186fcc1580c7b35ea60cae58c33fe4702fd1cc56e7ff12cb
Actual   6A SHA256: bdf56d8e972206b8186fcc1580c7b35ea60cae58c33fe4702fd1cc56e7ff12cb
6A source selection : NONE
6A Pareto computed  : NO

STAGE26-6B :: FROZEN SOURCE IDENTITY
measurement protocol                    PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
Stage26-0B resolved artifacts           PASS bb73dd1c63ab78942cea9eba9c1d1bc9d278cb3ae4963e8c7410b37ef8d270ad
Stage26-0B comparability groups         PASS 44d54cfa97360a094a6c86cb3e5d447875ebe79f0a0e5232b96b174abcf459ca
Stage26-0B candidate registry           PASS 610172f5324d81dccf40d536c9c4a7a22953404dfb67a4c5bf4d197610ca608a
Stage15 FT validation/holdout gap  

In [13]:
# =============================================================================
# STAGE26-6C
# FREEZE EXACT SIX CPU PARETO INPUT POINTS + GIT ANCHOR
#
# SCIENTIFIC PARENT:
#   92495eac2c9202973ff4d889e3c12669c06e9ef1
#
# INPUT FROM STAGE26-6B:
#   /kaggle/working/stage26_deployment_profiling/pareto/
#       stage26_6b_resolved_pareto_points.json
#
# EXPECTED 6B SHA256:
#   715c05709f865477aacc9546377f45f9484eefc9a55559506e3d42fac26a44a7
#
# EXPECTED SIX-POINT CORE SHA256:
#   5fb0e952d6560b6cd0f84c49048c05d26ff3c89c52edd61ea91855d374c10a76
#
# PURPOSE
# -------
# Freeze the six resolved Stage26 Pareto input points BEFORE any Pareto
# dominance/frontier computation.
#
# THIS CELL DOES:
#   - verify durable parent
#   - verify Stage26-6B transient resolution identity
#   - verify exact six models, groups, PR-AUC values and CPU1/B1 p95 values
#   - verify frozen Pareto rules from measurement_protocol.json
#   - persist immutable point-input lock
#   - persist Pareto-rule lock
#   - persist freeze receipt + manifest
#   - commit
#   - push
#   - remotely byte-verify and scientifically verify
#
# THIS CELL DOES NOT:
#   - compute dominance
#   - compute frontier membership
#   - compute bootstrap uncertainty
#   - recompute PR-AUC
#   - reopen holdout
#   - load models
#   - run inference
#   - perform timing
#   - access PCAP
#   - access Release corpus
#   - use GPU
# =============================================================================

from __future__ import annotations

import os
import json
import stat
import hashlib
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "92495eac2c9202973ff4d889e3c12669c06e9ef1"
)

COMMIT_SUBJECT = (
    "stage26: freeze CPU Pareto point inputs"
)


RUNTIME_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

SOURCE_6B = (
    RUNTIME_ROOT
    / "pareto"
    / "stage26_6b_resolved_pareto_points.json"
)

EXPECTED_6B_SHA256 = (
    "715c05709f865477aacc9546377f45f9484eefc9a55559506e3d42fac26a44a7"
)

EXPECTED_POINT_CORE_SHA256 = (
    "5fb0e952d6560b6cd0f84c49048c05d26ff3c89c52edd61ea91855d374c10a76"
)


PROTOCOL = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)


CHECKPOINT_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_6c_pareto_point_lock"
)

CHECKPOINT_DIR = (
    REPO
    / CHECKPOINT_REL
)

POINT_LOCK_PATH = (
    CHECKPOINT_DIR
    / "stage26_6c_resolved_pareto_points_lock.json"
)

RULE_LOCK_PATH = (
    CHECKPOINT_DIR
    / "stage26_6c_pareto_rules_lock.json"
)

RECEIPT_PATH = (
    CHECKPOINT_DIR
    / "stage26_6c_pareto_point_freeze_receipt.json"
)

MANIFEST_PATH = (
    CHECKPOINT_DIR
    / "stage26_6c_pareto_point_lock_manifest.json"
)


# =============================================================================
# 1. EXACT EXPECTED SIX POINTS
# =============================================================================

EXPECTED_POINTS = {
    "STAGE16_XGBOOST_TUNED": {
        "comparison_group":
            "GROUP_A_DUPSAFE70",

        "pr_auc":
            0.9453847557659393,

        "p95_cpu1_batch1_inference_latency_ms":
            0.5937109499999996,

        "condition_id":
            "CPUCOND_001",
    },

    "STAGE16_LIGHTGBM_TUNED": {
        "comparison_group":
            "GROUP_A_DUPSAFE70",

        "pr_auc":
            0.9466063189077636,

        "p95_cpu1_batch1_inference_latency_ms":
            0.64876045,

        "condition_id":
            "CPUCOND_011",
    },

    "STAGE16_CATBOOST_TUNED": {
        "comparison_group":
            "GROUP_A_DUPSAFE70",

        "pr_auc":
            0.9430447292850749,

        "p95_cpu1_batch1_inference_latency_ms":
            0.23919549999999995,

        "condition_id":
            "CPUCOND_021",
    },

    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING": {
        "comparison_group":
            "GROUP_A_DUPSAFE70",

        "pr_auc":
            0.9295715928199866,

        "p95_cpu1_batch1_inference_latency_ms":
            11.302716,

        "condition_id":
            "CPUCOND_031",
    },

    "STAGE20_MASKED_CNN_V1": {
        "comparison_group":
            "GROUP_B_PACKET_IMAGE",

        "pr_auc":
            0.48945269459245255,

        "p95_cpu1_batch1_inference_latency_ms":
            9.5320395,

        "condition_id":
            "CPUCOND_041",
    },

    "STAGE21_MASKED_VIT_V1": {
        "comparison_group":
            "GROUP_B_PACKET_IMAGE",

        "pr_auc":
            0.606536911289453,

        "p95_cpu1_batch1_inference_latency_ms":
            2.1051985,

        "condition_id":
            "CPUCOND_051",
    },
}


EXPECTED_ORDER = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
]


# =============================================================================
# 2. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        +
        "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8
                *
                1024
                *
                1024
            )

            if not block:

                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        +
        ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        text=False,
    )

    return bytes(
        p.stdout
    )


# =============================================================================
# 3. DURABLE GIT GATE
# =============================================================================

banner(
    "STAGE26-6C :: DURABLE SCIENTIFIC PARENT"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-6C parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Pareto input freeze."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if CHECKPOINT_DIR.exists():

    raise RuntimeError(
        "Stage26-6C checkpoint already exists."
    )


# =============================================================================
# 4. STAGE26-6B INPUT IDENTITY
# =============================================================================

banner(
    "STAGE26-6C :: STAGE26-6B INPUT IDENTITY"
)


if not SOURCE_6B.is_file():

    raise FileNotFoundError(
        SOURCE_6B
    )


source_6b_sha = sha256_file(
    SOURCE_6B
)


print(
    "Expected 6B SHA256:",
    EXPECTED_6B_SHA256
)

print(
    "Actual   6B SHA256:",
    source_6b_sha
)


if source_6b_sha != EXPECTED_6B_SHA256:

    raise RuntimeError(
        "Stage26-6B resolved point file mismatch."
    )


resolved = json.loads(
    SOURCE_6B.read_text(
        encoding="utf-8"
    )
)


if resolved[
    "scientific_parent"
] != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-6B scientific parent mismatch."
    )


if resolved[
    "point_set_sha256"
] != EXPECTED_POINT_CORE_SHA256:

    raise RuntimeError(
        "Six-point core fingerprint mismatch."
    )


science = resolved[
    "scientific_state"
]


required_false = [
    "predictive_metric_recomputed",
    "holdout_reopened",
    "model_loaded",
    "inference_performed",
    "timing_performed",
    "bootstrap_computed",
    "dominance_computed",
    "pareto_frontier_computed",
    "cross_group_comparison",
    "operational_reference_added_as_primary",
    "ft_single_resource_reference_added_to_pareto",
    "pcap_accessed",
    "release_corpus_accessed",
    "gpu_used",
    "git_modified",
]


for key in required_false:

    if science[
        key
    ] is not False:

        raise RuntimeError(
            f"Stage26-6B scientific boundary failed: {key}"
        )


if science[
    "six_primary_points_resolved"
] is not True:

    raise RuntimeError(
        "Stage26-6B did not declare six resolved primary points."
    )


print(
    "Six primary points resolved : PASS"
)

print(
    "Dominance computed in 6B    : NO"
)

print(
    "Frontier computed in 6B     : NO"
)

print(
    "GPU used in 6B              : NO"
)


# =============================================================================
# 5. EXACT SIX-POINT CONTENT GATE
# =============================================================================

banner(
    "STAGE26-6C :: EXACT SIX-POINT CONTENT GATE"
)


points = resolved[
    "points"
]


if len(
    points
) != 6:

    raise RuntimeError(
        f"Expected 6 primary points, found {len(points)}."
    )


actual_order = [
    point[
        "target_id"
    ]
    for point in points
]


if actual_order != EXPECTED_ORDER:

    raise RuntimeError(
        "Resolved six-point order changed."
    )


for point in points:

    target = point[
        "target_id"
    ]

    expected = EXPECTED_POINTS[
        target
    ]


    checks = {
        "group":
            (
                point[
                    "comparison_group"
                ]
                ==
                expected[
                    "comparison_group"
                ]
            ),

        "PR-AUC":
            (
                float(
                    point[
                        "pr_auc"
                    ]
                )
                ==
                float(
                    expected[
                        "pr_auc"
                    ]
                )
            ),

        "CPU1/B1 p95":
            (
                float(
                    point[
                        "p95_cpu1_batch1_inference_latency_ms"
                    ]
                )
                ==
                float(
                    expected[
                        "p95_cpu1_batch1_inference_latency_ms"
                    ]
                )
            ),

        "condition":
            (
                point[
                    "cost_source"
                ][
                    "condition_id"
                ]
                ==
                expected[
                    "condition_id"
                ]
            ),

        "hardware":
            (
                point[
                    "cost_source"
                ][
                    "hardware_mode"
                ]
                ==
                "CPU_1_PHYSICAL_CORE"
            ),

        "thread_count":
            (
                int(
                    point[
                        "cost_source"
                    ][
                        "thread_count"
                    ]
                )
                ==
                1
            ),

        "batch_size":
            (
                int(
                    point[
                        "cost_source"
                    ][
                        "batch_size"
                    ]
                )
                ==
                1
            ),

        "timed_runs":
            (
                int(
                    point[
                        "cost_source"
                    ][
                        "timed_runs"
                    ]
                )
                ==
                200
            ),

        "condition_PASS":
            (
                point[
                    "cost_source"
                ][
                    "status"
                ]
                ==
                "PASS"
            ),
    }


    print(
        "\n",
        target,
        sep="",
    )


    for label, passed in checks.items():

        print(
            f"  {label:18s}: "
            f"{'PASS' if passed else 'FAIL'}"
        )


    if not all(
        checks.values()
    ):

        raise RuntimeError(
            f"{target}: six-point freeze gate failed."
        )


    print(
        "  PR-AUC            :",
        repr(
            float(
                point[
                    "pr_auc"
                ]
            )
        )
    )

    print(
        "  CPU1 B1 p95 ms    :",
        repr(
            float(
                point[
                    "p95_cpu1_batch1_inference_latency_ms"
                ]
            )
        )
    )

    print(
        "  condition         :",
        point[
            "cost_source"
        ][
            "condition_id"
        ]
    )


# =============================================================================
# 6. RECOMPUTE CORE FINGERPRINT FROM RESOLVED FILE ONLY
# =============================================================================

banner(
    "STAGE26-6C :: SIX-POINT CORE FINGERPRINT"
)


point_core = [
    {
        "target_id":
            item[
                "target_id"
            ],

        "comparison_group":
            item[
                "comparison_group"
            ],

        "pr_auc":
            item[
                "pr_auc"
            ],

        "p95_cpu1_batch1_inference_latency_ms":
            item[
                "p95_cpu1_batch1_inference_latency_ms"
            ],
    }
    for item in points
]


point_core_bytes = json.dumps(
    point_core,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
    allow_nan=False,
).encode(
    "utf-8"
)


recomputed_core_sha = hashlib.sha256(
    point_core_bytes
).hexdigest()


print(
    "Expected:",
    EXPECTED_POINT_CORE_SHA256
)

print(
    "Actual  :",
    recomputed_core_sha
)


if recomputed_core_sha != EXPECTED_POINT_CORE_SHA256:

    raise RuntimeError(
        "Six-point core fingerprint failed recomputation."
    )


# =============================================================================
# 7. FROZEN PARETO PROTOCOL GATE
# =============================================================================

banner(
    "STAGE26-6C :: FROZEN PARETO RULES"
)


protocol_sha = sha256_file(
    PROTOCOL
)


print(
    "Protocol SHA256:"
)

print(
    " expected:",
    EXPECTED_PROTOCOL_SHA256
)

print(
    " actual  :",
    protocol_sha
)


if protocol_sha != EXPECTED_PROTOCOL_SHA256:

    raise RuntimeError(
        "Measurement protocol identity mismatch."
    )


protocol = json.loads(
    PROTOCOL.read_text(
        encoding="utf-8"
    )
)


pareto = protocol[
    "pareto_protocol"
]


group_a = pareto[
    "GROUP_A_DUPSAFE70"
]

group_b = pareto[
    "GROUP_B_PACKET_IMAGE"
]

dominance_rule = pareto[
    "dominance_rule"
]

operational_refs = pareto[
    "operational_references"
]


expected_group_a = EXPECTED_ORDER[
    :4
]

expected_group_b = EXPECTED_ORDER[
    4:
]


if group_a[
    "eligible_primary_members"
] != expected_group_a:

    raise RuntimeError(
        "Frozen Group-A primary universe changed."
    )


if group_b[
    "eligible_primary_members"
] != expected_group_b:

    raise RuntimeError(
        "Frozen Group-B primary universe changed."
    )


if group_a[
    "primary_cost_axis"
] != (
    "CPU_1_PHYSICAL_CORE batch1 p95 inference latency"
):

    raise RuntimeError(
        "Frozen Group-A cost axis changed."
    )


if group_b[
    "primary_cost_axis"
] != (
    "CPU_1_PHYSICAL_CORE batch1 p95 inference latency"
):

    raise RuntimeError(
        "Frozen Group-B cost axis changed."
    )


if group_a[
    "primary_discrimination_axis"
] != (
    "Same-population frozen PR-AUC"
):

    raise RuntimeError(
        "Frozen Group-A discrimination axis changed."
    )


if group_b[
    "primary_discrimination_axis"
] != (
    "Same-Friday-population frozen PR-AUC"
):

    raise RuntimeError(
        "Frozen Group-B discrimination axis changed."
    )


if group_b[
    "claim_boundary"
] != "DESCRIPTIVE_NON_CONFIRMATORY":

    raise RuntimeError(
        "Frozen Group-B claim boundary changed."
    )


if pareto[
    "global_rule"
] != "NO CROSS-GROUP PARETO FRONTIER.":

    raise RuntimeError(
        "Frozen global Pareto rule changed."
    )


if dominance_rule[
    "A_dominates_B_if"
] != (
    "A PR-AUC >= B PR-AUC and A p95 latency <= B p95 latency, "
    "with at least one strict inequality."
):

    raise RuntimeError(
        "Frozen dominance rule changed."
    )


if dominance_rule[
    "frontier_status"
] != (
    "Descriptive point-estimate frontier; bootstrap uncertainty "
    "reported separately."
):

    raise RuntimeError(
        "Frozen frontier-status rule changed."
    )


if (
    "does not increase architecture-family candidate count"
    not in
    operational_refs[
        "ENS_LGBM_XGB_EQUAL"
    ]
):

    raise RuntimeError(
        "Operational ensemble role changed."
    )


if (
    "excluded from predictive Pareto"
    not in
    operational_refs[
        "FT_BALANCED_SINGLE_RESOURCE_REFERENCE"
    ]
):

    raise RuntimeError(
        "FT single reference exclusion changed."
    )


print(
    "Group A primary members : PASS"
)

print(
    "Group B primary members : PASS"
)

print(
    "Cost axis               : CPU1 B=1 p95"
)

print(
    "Group A metric          : same-population frozen PR-AUC"
)

print(
    "Group B metric          : same-Friday frozen PR-AUC"
)

print(
    "Group B claim           : DESCRIPTIVE_NON_CONFIRMATORY"
)

print(
    "Cross-group frontier    : FORBIDDEN"
)

print(
    "Dominance calculation   : NOT EXECUTED"
)

print(
    "Bootstrap uncertainty   : NOT EXECUTED"
)


# =============================================================================
# 8. CREATE DURABLE LOCK PACKAGE
# =============================================================================

banner(
    "STAGE26-6C :: WRITE DURABLE POINT LOCK"
)


CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


point_lock = {
    "schema":
        "stage26_6c_resolved_pareto_points_lock_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-6C",

    "status":
        "FROZEN_BEFORE_DOMINANCE",

    "scientific_parent":
        EXPECTED_PARENT,

    "source_stage26_6b": {
        "path":
            str(
                SOURCE_6B
            ),

        "sha256":
            EXPECTED_6B_SHA256,

        "point_set_sha256":
            EXPECTED_POINT_CORE_SHA256,
    },

    "point_count":
        6,

    "point_order":
        EXPECTED_ORDER,

    "points":
        points,

    "point_set_sha256":
        EXPECTED_POINT_CORE_SHA256,

    "group_partition": {
        "GROUP_A_DUPSAFE70":
            expected_group_a,

        "GROUP_B_PACKET_IMAGE":
            expected_group_b,
    },

    "source_resolution_policy": {
        "GROUP_A_DUPSAFE70":
            (
                "Frozen PR-AUC values share the exact duplicate-safe "
                "validation population inherited by Stage15/Stage16."
            ),

        "GROUP_B_PACKET_IMAGE":
            (
                "Frozen PR-AUC values share the exact same Friday compact "
                "corpus export order and remain descriptive/non-confirmatory."
            ),

        "cost_axis":
            (
                "CPU_1_PHYSICAL_CORE batch1 p95 isolated inference latency."
            ),
    },

    "excluded_from_primary_point_set": {
        "ENS_LGBM_XGB_EQUAL":
            (
                "Operational reference only; not an additional architecture "
                "family primary point."
            ),

        "FT_BALANCED_SINGLE_RESOURCE_REFERENCE":
            (
                "Resource decomposition only; excluded from predictive Pareto."
            ),
    },

    "scientific_state_at_freeze": {
        "dominance_computed":
            False,

        "frontier_computed":
            False,

        "bootstrap_uncertainty_computed":
            False,

        "cross_group_frontier_computed":
            False,

        "predictive_metric_recomputed":
            False,

        "holdout_reopened":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "pcap_accessed":
            False,

        "release_corpus_accessed":
            False,

        "gpu_used":
            False,
    },
}


rule_lock = {
    "schema":
        "stage26_6c_pareto_rules_lock_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-6C",

    "measurement_protocol_path":
        str(
            PROTOCOL.relative_to(
                REPO
            )
        ),

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "GROUP_A_DUPSAFE70":
        group_a,

    "GROUP_B_PACKET_IMAGE":
        group_b,

    "dominance_rule":
        dominance_rule,

    "global_rule":
        pareto[
            "global_rule"
        ],

    "operational_references":
        operational_refs,

    "execution_guard": {
        "point_estimate_frontier_may_be_computed_only_after_this_lock_is_git_anchored":
            True,

        "bootstrap_uncertainty_is_separate_from_point_estimate_frontier":
            True,

        "cross_group_frontier_prohibited":
            True,

        "post_freeze_metric_substitution_prohibited":
            True,

        "post_freeze_cost_axis_substitution_prohibited":
            True,
    },
}


atomic_json(
    POINT_LOCK_PATH,
    point_lock,
)

atomic_json(
    RULE_LOCK_PATH,
    rule_lock,
)


point_lock_sha = sha256_file(
    POINT_LOCK_PATH
)

rule_lock_sha = sha256_file(
    RULE_LOCK_PATH
)


receipt = {
    "schema":
        "stage26_6c_pareto_point_freeze_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-6C",

    "status":
        "PASS_SIX_PRIMARY_PARETO_POINTS_FROZEN_BEFORE_DOMINANCE",

    "scientific_parent":
        EXPECTED_PARENT,

    "source_6b_sha256":
        EXPECTED_6B_SHA256,

    "point_set_sha256":
        EXPECTED_POINT_CORE_SHA256,

    "point_lock_sha256":
        point_lock_sha,

    "rule_lock_sha256":
        rule_lock_sha,

    "point_count":
        6,

    "group_A_point_count":
        4,

    "group_B_point_count":
        2,

    "dominance_computed":
        False,

    "frontier_computed":
        False,

    "bootstrap_uncertainty_computed":
        False,

    "cross_group_frontier_computed":
        False,

    "scientific_boundaries": {
        "PR_AUC_recomputed":
            False,

        "holdout_reopened":
            False,

        "models_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "PCAP_accessed":
            False,

        "Release_corpus_accessed":
            False,

        "GPU_used":
            False,
    },

    "next_permitted_action":
        (
            "After remote Git verification, compute within-group "
            "point-estimate Pareto dominance using only the frozen six points."
        ),
}


atomic_json(
    RECEIPT_PATH,
    receipt,
)


receipt_sha = sha256_file(
    RECEIPT_PATH
)


# =============================================================================
# 9. PACKAGE MANIFEST
# =============================================================================

package_files = [
    POINT_LOCK_PATH,
    RULE_LOCK_PATH,
    RECEIPT_PATH,
]


manifest_rows = []


for path in package_files:

    manifest_rows.append(
        {
            "repo_relative_path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest = {
    "schema":
        "stage26_6c_pareto_point_lock_manifest_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        "READY_FOR_GIT_ANCHOR",

    "scientific_parent":
        EXPECTED_PARENT,

    "commit_subject":
        COMMIT_SUBJECT,

    "source_stage26_6b_sha256":
        EXPECTED_6B_SHA256,

    "point_set_sha256":
        EXPECTED_POINT_CORE_SHA256,

    "point_lock_sha256":
        point_lock_sha,

    "rule_lock_sha256":
        rule_lock_sha,

    "receipt_sha256":
        receipt_sha,

    "file_count_excluding_manifest":
        len(
            manifest_rows
        ),

    "files":
        manifest_rows,

    "scientific_state": {
        "point_inputs_frozen":
            True,

        "dominance_computed":
            False,

        "frontier_computed":
            False,

        "bootstrap_uncertainty_computed":
            False,

        "cross_group_frontier_computed":
            False,

        "new_measurement_performed":
            False,

        "GPU_used":
            False,
    },
}


atomic_json(
    MANIFEST_PATH,
    manifest,
)


manifest_sha = sha256_file(
    MANIFEST_PATH
)


print(
    "Point lock:",
    point_lock_sha
)

print(
    "Rule lock :",
    rule_lock_sha
)

print(
    "Receipt   :",
    receipt_sha
)

print(
    "Manifest  :",
    manifest_sha
)


# =============================================================================
# 10. LOCAL PACKAGE BYTE AUDIT
# =============================================================================

banner(
    "STAGE26-6C :: LOCAL PACKAGE AUDIT"
)


for row in manifest_rows:

    path = (
        REPO
        / row[
            "repo_relative_path"
        ]
    )

    actual_size = int(
        path.stat().st_size
    )

    actual_sha = sha256_file(
        path
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Local Stage26-6C byte audit failed."
        )


# =============================================================================
# 11. GIT CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-6C :: GIT CHANGE AUDIT"
)


repo_status = git(
    "status",
    "--porcelain",
)


print(
    repo_status
)


if not repo_status:

    raise RuntimeError(
        "Expected uncommitted Stage26-6C package."
    )


unexpected = []


for line in repo_status.splitlines():

    relpath = line[
        3:
    ]


    if not relpath.startswith(
        str(
            CHECKPOINT_REL
        )
        +
        "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository changes:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 12. GIT IDENTITY
# =============================================================================

author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_PARENT,
)

author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_PARENT,
)


git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


print(
    "\nGit author:"
)

print(
    " ",
    author_name,
    "<" + author_email + ">"
)


# =============================================================================
# 13. COMMIT
# =============================================================================

banner(
    "STAGE26-6C :: COMMIT"
)


git(
    "add",
    str(
        CHECKPOINT_REL
    ),
)


print(
    git(
        "diff",
        "--cached",
        "--name-status",
    )
)


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-6C commit parent mismatch."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Stage26-6C commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository not clean after Stage26-6C commit."
    )


# =============================================================================
# 14. PUSH
# =============================================================================

banner(
    "STAGE26-6C :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    result = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
        text=True,
    )


    print(
        result.stdout.strip()
    )


github_token = None


# =============================================================================
# 15. REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-6C :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "Remote did not advance to Stage26-6C commit."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote Stage26-6C parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote Stage26-6C subject mismatch."
    )


# =============================================================================
# 16. REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-6C :: REMOTE BYTE VERIFICATION"
)


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    str(
        MANIFEST_PATH.relative_to(
            REPO
        )
    ),
)

remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "Remote manifest SHA256:"
)

print(
    " ",
    remote_manifest_sha
)


if remote_manifest_sha != manifest_sha:

    raise RuntimeError(
        "Remote Stage26-6C manifest SHA mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


for row in remote_manifest[
    "files"
]:

    data = git_blob_bytes(
        "origin/main",
        row[
            "repo_relative_path"
        ],
    )

    actual_size = len(
        data
    )

    actual_sha = sha256_bytes(
        data
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Remote Stage26-6C byte verification failed."
        )


# =============================================================================
# 17. REMOTE SCIENTIFIC VERIFICATION
# =============================================================================

banner(
    "STAGE26-6C :: REMOTE SCIENTIFIC VERIFICATION"
)


remote_lock = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            POINT_LOCK_PATH.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_rules = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            RULE_LOCK_PATH.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_receipt = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            RECEIPT_PATH.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)


remote_points = remote_lock[
    "points"
]


remote_checks = {
    "six points":
        (
            len(
                remote_points
            )
            ==
            6
        ),

    "point fingerprint":
        (
            remote_lock[
                "point_set_sha256"
            ]
            ==
            EXPECTED_POINT_CORE_SHA256
        ),

    "frozen before dominance":
        (
            remote_lock[
                "status"
            ]
            ==
            "FROZEN_BEFORE_DOMINANCE"
        ),

    "dominance false":
        (
            remote_receipt[
                "dominance_computed"
            ]
            is False
        ),

    "frontier false":
        (
            remote_receipt[
                "frontier_computed"
            ]
            is False
        ),

    "bootstrap false":
        (
            remote_receipt[
                "bootstrap_uncertainty_computed"
            ]
            is False
        ),

    "cross-group false":
        (
            remote_receipt[
                "cross_group_frontier_computed"
            ]
            is False
        ),

    "Group A 4 points":
        (
            remote_receipt[
                "group_A_point_count"
            ]
            ==
            4
        ),

    "Group B 2 points":
        (
            remote_receipt[
                "group_B_point_count"
            ]
            ==
            2
        ),

    "global no-cross rule":
        (
            remote_rules[
                "global_rule"
            ]
            ==
            "NO CROSS-GROUP PARETO FRONTIER."
        ),

    "GPU false":
        (
            remote_receipt[
                "scientific_boundaries"
            ][
                "GPU_used"
            ]
            is False
        ),
}


for name, passed in remote_checks.items():

    print(
        f"{name:32s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    remote_checks.values()
):

    raise RuntimeError(
        "Remote Stage26-6C scientific verification failed."
    )


# Verify every remote point exactly.
for remote_point in remote_points:

    target = remote_point[
        "target_id"
    ]

    expected = EXPECTED_POINTS[
        target
    ]


    if (
        remote_point[
            "comparison_group"
        ]
        !=
        expected[
            "comparison_group"
        ]
    ):

        raise RuntimeError(
            f"Remote group mismatch for {target}."
        )


    if (
        float(
            remote_point[
                "pr_auc"
            ]
        )
        !=
        float(
            expected[
                "pr_auc"
            ]
        )
    ):

        raise RuntimeError(
            f"Remote PR-AUC mismatch for {target}."
        )


    if (
        float(
            remote_point[
                "p95_cpu1_batch1_inference_latency_ms"
            ]
        )
        !=
        float(
            expected[
                "p95_cpu1_batch1_inference_latency_ms"
            ]
        )
    ):

        raise RuntimeError(
            f"Remote p95 mismatch for {target}."
        )


    if (
        remote_point[
            "cost_source"
        ][
            "condition_id"
        ]
        !=
        expected[
            "condition_id"
        ]
    ):

        raise RuntimeError(
            f"Remote condition mismatch for {target}."
        )


    print(
        f"PASS {target:42s} "
        f"PR={float(remote_point['pr_auc']):.15f} "
        f"p95={float(remote_point['p95_cpu1_batch1_inference_latency_ms']):.12f}"
    )


# =============================================================================
# 18. FINAL CLOSURE
# =============================================================================

banner(
    "STAGE26-6C PARETO INPUT FREEZE COMPLETE"
)


final_status = git(
    "status",
    "--porcelain",
)


print(
    "NEW DURABLE COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nPARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nFROZEN POINT CORE SHA256:"
)

print(
    " ",
    EXPECTED_POINT_CORE_SHA256
)


print(
    "\nPACKAGE HASHES:"
)

print(
    "  point lock:",
    point_lock_sha
)

print(
    "  rule lock :",
    rule_lock_sha
)

print(
    "  receipt   :",
    receipt_sha
)

print(
    "  manifest  :",
    manifest_sha
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  six primary points frozen      : YES"
)

print(
    "  dominance computed             : NO"
)

print(
    "  frontier computed              : NO"
)

print(
    "  bootstrap uncertainty computed : NO"
)

print(
    "  cross-group frontier           : NO"
)

print(
    "  new predictive evaluation      : NO"
)

print(
    "  new inference                  : NO"
)

print(
    "  new timing                     : NO"
)

print(
    "  holdout reopened               : NO"
)

print(
    "  GPU                            : NO"
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  commit             : PASS"
)

print(
    "  parent             : PASS"
)

print(
    "  package bytes      : PASS"
)

print(
    "  exact six points   : PASS"
)

print(
    "  scientific content : PASS"
)


print(
    "\nRepo clean:",
    final_status == ""
)


if final_status:

    raise RuntimeError(
        "Repository not clean after Stage26-6C."
    )


print(
    "\nNEXT:"
)

print(
    "  Stage26-6D may now compute the frozen within-group"
)

print(
    "  point-estimate Pareto dominance/frontiers."
)

print(
    "  Group A and Group B MUST remain separate."
)

print(
    "  Group B remains descriptive/non-confirmatory."
)

print(
    "  Bootstrap uncertainty remains a separate step."
)

print(
    "  No GPU yet."
)


STAGE26-6C :: DURABLE SCIENTIFIC PARENT
Expected parent: 92495eac2c9202973ff4d889e3c12669c06e9ef1
Local HEAD     : 92495eac2c9202973ff4d889e3c12669c06e9ef1
origin/main    : 92495eac2c9202973ff4d889e3c12669c06e9ef1
Repo clean     : True

STAGE26-6C :: STAGE26-6B INPUT IDENTITY
Expected 6B SHA256: 715c05709f865477aacc9546377f45f9484eefc9a55559506e3d42fac26a44a7
Actual   6B SHA256: 715c05709f865477aacc9546377f45f9484eefc9a55559506e3d42fac26a44a7
Six primary points resolved : PASS
Dominance computed in 6B    : NO
Frontier computed in 6B     : NO
GPU used in 6B              : NO

STAGE26-6C :: EXACT SIX-POINT CONTENT GATE

STAGE16_XGBOOST_TUNED
  group             : PASS
  PR-AUC            : PASS
  CPU1/B1 p95       : PASS
  condition         : PASS
  hardware          : PASS
  thread_count      : PASS
  batch_size        : PASS
  timed_runs        : PASS
  condition_PASS    : PASS
  PR-AUC            : 0.9453847557659393
  CPU1 B1 p95 ms    : 0.5937109499999996
  condition         : CPUC

RuntimeError: STAGE16_LIGHTGBM_TUNED: six-point freeze gate failed.

In [14]:
# =============================================================================
# STAGE26-6C-DIAG
# NARROW DIAGNOSIS OF LIGHTGBM CPU1/B1 P95 FLOAT MISMATCH
#
# NO WRITES
# NO DOMINANCE
# NO PARETO
# NO MEASUREMENT
# NO GPU
# =============================================================================

from __future__ import annotations

import csv
import json
import hashlib
import subprocess
from pathlib import Path


REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "92495eac2c9202973ff4d889e3c12669c06e9ef1"
)

SOURCE_6B = Path(
    "/kaggle/working/stage26_deployment_profiling/"
    "pareto/stage26_6b_resolved_pareto_points.json"
)

EXPECTED_6B_SHA256 = (
    "715c05709f865477aacc9546377f45f9484eefc9a55559506e3d42fac26a44a7"
)

WARM_SUMMARY = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_2_cpu_warm_inference"
    / "stage26_2_warm_summary.csv"
)

TARGET = (
    "STAGE16_LIGHTGBM_TUNED"
)

CONDITION_ID = (
    "CPUCOND_011"
)

# Literal used by failed Cell 34.
FAILED_LITERAL = (
    0.64876045
)


def banner(text):
    print("\n" + "=" * 124)
    print(text)
    print("=" * 124)


def git(*args):
    p = subprocess.run(
        ["git", *args],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(p.stdout)

    return p.stdout.strip()


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(8 * 1024 * 1024)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


# =============================================================================
# 1. DURABLE STATE
# =============================================================================

banner(
    "STAGE26-6C-DIAG :: DURABLE STATE"
)

git(
    "fetch",
    "origin",
    "main",
)

head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print("Expected parent:", EXPECTED_PARENT)
print("Local HEAD     :", head)
print("origin/main    :", remote)
print("Repo clean     :", status == "")


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "HEAD changed after failed Stage26-6C."
    )

if remote != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main changed after failed Stage26-6C."
    )

if status:
    raise RuntimeError(
        "Repository is not clean after failed Stage26-6C."
    )


checkpoint_dir = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_6c_pareto_point_lock"
)


print(
    "6C checkpoint exists:",
    checkpoint_dir.exists()
)


if checkpoint_dir.exists():
    raise RuntimeError(
        "Unexpected Stage26-6C checkpoint exists."
    )


# =============================================================================
# 2. VERIFY STAGE26-6B IDENTITY
# =============================================================================

banner(
    "STAGE26-6C-DIAG :: STAGE26-6B IDENTITY"
)


actual_6b_sha = sha256_file(
    SOURCE_6B
)


print(
    "Expected SHA256:",
    EXPECTED_6B_SHA256
)

print(
    "Actual SHA256  :",
    actual_6b_sha
)


if actual_6b_sha != EXPECTED_6B_SHA256:
    raise RuntimeError(
        "Stage26-6B source identity changed."
    )


resolved = json.loads(
    SOURCE_6B.read_text(
        encoding="utf-8"
    )
)


# =============================================================================
# 3. EXTRACT EXACT LIGHTGBM VALUE FROM 6B
# =============================================================================

banner(
    "STAGE26-6C-DIAG :: LIGHTGBM EXACT FLOAT"
)


matches = [
    point
    for point in resolved["points"]
    if point["target_id"] == TARGET
]


if len(matches) != 1:
    raise RuntimeError(
        f"Expected one {TARGET} point; found {len(matches)}."
    )


point = matches[0]

actual = float(
    point[
        "p95_cpu1_batch1_inference_latency_ms"
    ]
)


print("Target             :", TARGET)
print("Condition          :", point["cost_source"]["condition_id"])
print()
print("6B JSON value repr :", repr(actual))
print("6B JSON value hex  :", actual.hex())
print()
print("Failed literal repr:", repr(FAILED_LITERAL))
print("Failed literal hex :", FAILED_LITERAL.hex())
print()
print("Exact equality     :", actual == FAILED_LITERAL)
print("Difference         :", repr(actual - FAILED_LITERAL))


if point[
    "cost_source"
][
    "condition_id"
] != CONDITION_ID:
    raise RuntimeError(
        "Unexpected LightGBM condition ID."
    )


# =============================================================================
# 4. READ ORIGINAL CSV STRING WITHOUT RECOMPUTATION
# =============================================================================

banner(
    "STAGE26-6C-DIAG :: ORIGINAL STAGE26-2 CSV FIELD"
)


with WARM_SUMMARY.open(
    "r",
    encoding="utf-8",
    newline="",
) as f:

    rows = list(
        csv.DictReader(f)
    )


csv_matches = [
    row
    for row in rows
    if (
        row.get("target_id") == TARGET
        and
        row.get("condition_id") == CONDITION_ID
        and
        row.get("hardware_mode") == "CPU_1_PHYSICAL_CORE"
        and
        row.get("batch_size") == "1"
    )
]


if len(csv_matches) != 1:
    raise RuntimeError(
        f"Expected one original LightGBM CSV row; "
        f"found {len(csv_matches)}."
    )


csv_row = csv_matches[0]

raw_string = csv_row[
    "p95_batch_latency_ms"
]

parsed_csv = float(
    raw_string
)


print(
    "Raw CSV field      :",
    repr(raw_string)
)

print(
    "Parsed CSV repr    :",
    repr(parsed_csv)
)

print(
    "Parsed CSV hex     :",
    parsed_csv.hex()
)

print(
    "CSV == 6B JSON     :",
    parsed_csv == actual
)


# =============================================================================
# 5. PRINT ALL SIX EXACT COST FLOATS
# =============================================================================

banner(
    "STAGE26-6C-DIAG :: ALL SIX EXACT CPU1/B1 P95 VALUES"
)


for point in resolved[
    "points"
]:

    value = float(
        point[
            "p95_cpu1_batch1_inference_latency_ms"
        ]
    )

    print(
        f"{point['target_id']:42s} "
        f"repr={repr(value):22s} "
        f"hex={value.hex()}"
    )


# =============================================================================
# 6. CLOSURE
# =============================================================================

banner(
    "STAGE26-6C-DIAG COMPLETE"
)


print(
    "Scientific data changed       : NO"
)

print(
    "Stage26-6B identity intact    : YES"
)

print(
    "Pareto dominance computed     : NO"
)

print(
    "Checkpoint written            : NO"
)

print(
    "Git modified                  : NO"
)

print(
    "GPU                           : NO"
)


print(
    "\nUse the exact 6B-stored float representation in the"
)

print(
    "narrow Stage26-6C recovery; do not relax equality and"
)

print(
    "do not alter any scientific point."
)


STAGE26-6C-DIAG :: DURABLE STATE
Expected parent: 92495eac2c9202973ff4d889e3c12669c06e9ef1
Local HEAD     : 92495eac2c9202973ff4d889e3c12669c06e9ef1
origin/main    : 92495eac2c9202973ff4d889e3c12669c06e9ef1
Repo clean     : True
6C checkpoint exists: False

STAGE26-6C-DIAG :: STAGE26-6B IDENTITY
Expected SHA256: 715c05709f865477aacc9546377f45f9484eefc9a55559506e3d42fac26a44a7
Actual SHA256  : 715c05709f865477aacc9546377f45f9484eefc9a55559506e3d42fac26a44a7

STAGE26-6C-DIAG :: LIGHTGBM EXACT FLOAT
Target             : STAGE16_LIGHTGBM_TUNED
Condition          : CPUCOND_011

6B JSON value repr : 0.6487604499999996
6B JSON value hex  : 0x1.4c2a546760613p-1

Failed literal repr: 0.64876045
Failed literal hex : 0x1.4c2a546760617p-1

Exact equality     : False
Difference         : -4.440892098500626e-16

STAGE26-6C-DIAG :: ORIGINAL STAGE26-2 CSV FIELD
Raw CSV field      : '0.6487604499999996'
Parsed CSV repr    : 0.6487604499999996
Parsed CSV hex     : 0x1.4c2a546760613p-1
CSV == 6B JSON   

In [15]:
# =============================================================================
# STAGE26-6C-R
# NARROW RECOVERY:
# FREEZE EXACT SIX CPU PARETO INPUT POINTS USING EXACT IEEE-754 IDENTITIES
#
# SCIENTIFIC PARENT:
#   92495eac2c9202973ff4d889e3c12669c06e9ef1
#
# WHY RECOVERY IS REQUIRED
# ------------------------
# Original Stage26-6C failed BEFORE ANY WRITE because several manually typed
# decimal literals did not have the exact same binary64 representation as the
# already-anchored Stage26-2 / Stage26-6B values.
#
# Diagnostic established:
#
# XGBoost:
#   0x1.2ffae1b30ddebp-1
#
# LightGBM:
#   0x1.4c2a546760613p-1
#
# CatBoost:
#   0x1.e9df548ecd8dap-3
#
# FT ensemble:
#   0x1.69afd976ff3aep+3
#
# CNN:
#   0x1.310677b395c41p+3
#
# ViT:
#   0x1.0d7724fa8b4bfp+1
#
# RECOVERY POLICY
# ---------------
# - preserve Stage26-6B bytes exactly
# - preserve six-point core SHA exactly
# - validate latency values by exact float.hex(), NOT tolerance
# - do NOT alter any point
# - do NOT compute dominance/frontier/bootstrap
#
# THIS CELL:
#   1. verifies durable parent / clean repo;
#   2. verifies Stage26-6B exact SHA + point-core SHA;
#   3. verifies all six groups / PR-AUC / condition IDs;
#   4. verifies CPU p95 values against exact IEEE-754 hex identities;
#   5. freezes Pareto point + rule locks;
#   6. commits + pushes;
#   7. remotely byte/scientifically verifies.
#
# NO:
#   - metric recomputation
#   - holdout reopening
#   - model loading
#   - inference
#   - timing
#   - dominance
#   - frontier
#   - bootstrap
#   - PCAP
#   - Release corpus
#   - GPU
# =============================================================================

from __future__ import annotations

import os
import json
import stat
import hashlib
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "92495eac2c9202973ff4d889e3c12669c06e9ef1"
)

COMMIT_SUBJECT = (
    "stage26: freeze CPU Pareto point inputs"
)


RUNTIME_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

SOURCE_6B = (
    RUNTIME_ROOT
    / "pareto"
    / "stage26_6b_resolved_pareto_points.json"
)

EXPECTED_6B_SHA256 = (
    "715c05709f865477aacc9546377f45f9484eefc9a55559506e3d42fac26a44a7"
)

EXPECTED_POINT_CORE_SHA256 = (
    "5fb0e952d6560b6cd0f84c49048c05d26ff3c89c52edd61ea91855d374c10a76"
)


PROTOCOL = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)


CHECKPOINT_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_6c_pareto_point_lock"
)

CHECKPOINT_DIR = (
    REPO
    / CHECKPOINT_REL
)

POINT_LOCK_PATH = (
    CHECKPOINT_DIR
    / "stage26_6c_resolved_pareto_points_lock.json"
)

RULE_LOCK_PATH = (
    CHECKPOINT_DIR
    / "stage26_6c_pareto_rules_lock.json"
)

RECEIPT_PATH = (
    CHECKPOINT_DIR
    / "stage26_6c_pareto_point_freeze_receipt.json"
)

MANIFEST_PATH = (
    CHECKPOINT_DIR
    / "stage26_6c_pareto_point_lock_manifest.json"
)


# =============================================================================
# 1. EXACT EXPECTED IDENTITIES
# =============================================================================

EXPECTED_ORDER = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
]


EXPECTED_POINTS = {
    "STAGE16_XGBOOST_TUNED": {
        "comparison_group":
            "GROUP_A_DUPSAFE70",

        "pr_auc":
            0.9453847557659393,

        "latency_hex":
            "0x1.2ffae1b30ddebp-1",

        "condition_id":
            "CPUCOND_001",
    },

    "STAGE16_LIGHTGBM_TUNED": {
        "comparison_group":
            "GROUP_A_DUPSAFE70",

        "pr_auc":
            0.9466063189077636,

        "latency_hex":
            "0x1.4c2a546760613p-1",

        "condition_id":
            "CPUCOND_011",
    },

    "STAGE16_CATBOOST_TUNED": {
        "comparison_group":
            "GROUP_A_DUPSAFE70",

        "pr_auc":
            0.9430447292850749,

        "latency_hex":
            "0x1.e9df548ecd8dap-3",

        "condition_id":
            "CPUCOND_021",
    },

    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING": {
        "comparison_group":
            "GROUP_A_DUPSAFE70",

        "pr_auc":
            0.9295715928199866,

        "latency_hex":
            "0x1.69afd976ff3aep+3",

        "condition_id":
            "CPUCOND_031",
    },

    "STAGE20_MASKED_CNN_V1": {
        "comparison_group":
            "GROUP_B_PACKET_IMAGE",

        "pr_auc":
            0.48945269459245255,

        "latency_hex":
            "0x1.310677b395c41p+3",

        "condition_id":
            "CPUCOND_041",
    },

    "STAGE21_MASKED_VIT_V1": {
        "comparison_group":
            "GROUP_B_PACKET_IMAGE",

        "pr_auc":
            0.606536911289453,

        "latency_hex":
            "0x1.0d7724fa8b4bfp+1",

        "condition_id":
            "CPUCOND_051",
    },
}


# =============================================================================
# 2. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        +
        "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8
                *
                1024
                *
                1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        +
        ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        text=False,
    )

    return bytes(
        p.stdout
    )


# =============================================================================
# 3. DURABLE STATE / FAILED-6C RECOVERY GATE
# =============================================================================

banner(
    "STAGE26-6C-R :: DURABLE STATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)

print(
    "6C checkpoint exists:",
    CHECKPOINT_DIR.exists()
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-6C recovery parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed after failed 6C attempt."
    )


if status:

    raise RuntimeError(
        "Repository is not clean."
    )


if CHECKPOINT_DIR.exists():

    raise RuntimeError(
        "Unexpected 6C checkpoint exists. "
        "Do not overwrite."
    )


print(
    "\n[PASS] Original 6C failed before any checkpoint write."
)


# =============================================================================
# 4. STAGE26-6B IDENTITY
# =============================================================================

banner(
    "STAGE26-6C-R :: STAGE26-6B IDENTITY"
)


if not SOURCE_6B.is_file():

    raise FileNotFoundError(
        SOURCE_6B
    )


source_sha = sha256_file(
    SOURCE_6B
)


print(
    "Expected 6B SHA256:",
    EXPECTED_6B_SHA256
)

print(
    "Actual   6B SHA256:",
    source_sha
)


if source_sha != EXPECTED_6B_SHA256:

    raise RuntimeError(
        "Stage26-6B source identity mismatch."
    )


resolved = json.loads(
    SOURCE_6B.read_text(
        encoding="utf-8"
    )
)


if resolved[
    "scientific_parent"
] != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-6B parent mismatch."
    )


if resolved[
    "point_set_sha256"
] != EXPECTED_POINT_CORE_SHA256:

    raise RuntimeError(
        "Stage26-6B point-core SHA mismatch."
    )


science = resolved[
    "scientific_state"
]


for key in [
    "predictive_metric_recomputed",
    "holdout_reopened",
    "model_loaded",
    "inference_performed",
    "timing_performed",
    "bootstrap_computed",
    "dominance_computed",
    "pareto_frontier_computed",
    "cross_group_comparison",
    "operational_reference_added_as_primary",
    "ft_single_resource_reference_added_to_pareto",
    "pcap_accessed",
    "release_corpus_accessed",
    "gpu_used",
    "git_modified",
]:

    if science[
        key
    ] is not False:

        raise RuntimeError(
            f"Unexpected 6B scientific state: {key}"
        )


print(
    "Point set SHA : PASS"
)

print(
    "Dominance     : NOT COMPUTED"
)

print(
    "Frontier      : NOT COMPUTED"
)

print(
    "Bootstrap     : NOT COMPUTED"
)


# =============================================================================
# 5. EXACT SIX-POINT CONTENT + IEEE-754 GATE
# =============================================================================

banner(
    "STAGE26-6C-R :: EXACT SIX-POINT BINARY GATE"
)


points = resolved[
    "points"
]


if len(
    points
) != 6:

    raise RuntimeError(
        f"Expected six points; found {len(points)}."
    )


actual_order = [
    point[
        "target_id"
    ]
    for point in points
]


if actual_order != EXPECTED_ORDER:

    raise RuntimeError(
        "Six-point order mismatch."
    )


for point in points:

    target = point[
        "target_id"
    ]

    expected = EXPECTED_POINTS[
        target
    ]

    latency = float(
        point[
            "p95_cpu1_batch1_inference_latency_ms"
        ]
    )

    latency_hex = latency.hex()


    checks = {
        "group":
            (
                point[
                    "comparison_group"
                ]
                ==
                expected[
                    "comparison_group"
                ]
            ),

        "PR-AUC exact":
            (
                float(
                    point[
                        "pr_auc"
                    ]
                )
                ==
                float(
                    expected[
                        "pr_auc"
                    ]
                )
            ),

        "p95 binary64 exact":
            (
                latency_hex
                ==
                expected[
                    "latency_hex"
                ]
            ),

        "condition ID":
            (
                point[
                    "cost_source"
                ][
                    "condition_id"
                ]
                ==
                expected[
                    "condition_id"
                ]
            ),

        "CPU1":
            (
                point[
                    "cost_source"
                ][
                    "hardware_mode"
                ]
                ==
                "CPU_1_PHYSICAL_CORE"
            ),

        "affinity":
            (
                point[
                    "cost_source"
                ][
                    "affinity"
                ]
                ==
                [
                    0
                ]
            ),

        "threads":
            (
                int(
                    point[
                        "cost_source"
                    ][
                        "thread_count"
                    ]
                )
                ==
                1
            ),

        "batch":
            (
                int(
                    point[
                        "cost_source"
                    ][
                        "batch_size"
                    ]
                )
                ==
                1
            ),

        "timed runs":
            (
                int(
                    point[
                        "cost_source"
                    ][
                        "timed_runs"
                    ]
                )
                ==
                200
            ),

        "condition PASS":
            (
                point[
                    "cost_source"
                ][
                    "status"
                ]
                ==
                "PASS"
            ),
    }


    print(
        "\n",
        target,
        sep="",
    )


    for label, passed in checks.items():

        print(
            f"  {label:20s}: "
            f"{'PASS' if passed else 'FAIL'}"
        )


    print(
        "  PR-AUC repr         :",
        repr(
            float(
                point[
                    "pr_auc"
                ]
            )
        )
    )

    print(
        "  p95 repr            :",
        repr(
            latency
        )
    )

    print(
        "  p95 binary64        :",
        latency_hex
    )

    print(
        "  expected binary64   :",
        expected[
            "latency_hex"
        ]
    )

    print(
        "  condition           :",
        point[
            "cost_source"
        ][
            "condition_id"
        ]
    )


    if not all(
        checks.values()
    ):

        raise RuntimeError(
            f"{target}: exact point identity gate failed."
        )


# =============================================================================
# 6. RECOMPUTE POINT-CORE SHA
# =============================================================================

banner(
    "STAGE26-6C-R :: POINT-CORE SHA RECOMPUTATION"
)


point_core = [
    {
        "target_id":
            point[
                "target_id"
            ],

        "comparison_group":
            point[
                "comparison_group"
            ],

        "pr_auc":
            point[
                "pr_auc"
            ],

        "p95_cpu1_batch1_inference_latency_ms":
            point[
                "p95_cpu1_batch1_inference_latency_ms"
            ],
    }
    for point in points
]


core_bytes = json.dumps(
    point_core,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
    allow_nan=False,
).encode(
    "utf-8"
)


core_sha = hashlib.sha256(
    core_bytes
).hexdigest()


print(
    "Expected:",
    EXPECTED_POINT_CORE_SHA256
)

print(
    "Actual  :",
    core_sha
)


if core_sha != EXPECTED_POINT_CORE_SHA256:

    raise RuntimeError(
        "Exact six-point core SHA recomputation failed."
    )


# =============================================================================
# 7. FROZEN PARETO PROTOCOL
# =============================================================================

banner(
    "STAGE26-6C-R :: FROZEN PARETO RULES"
)


protocol_sha = sha256_file(
    PROTOCOL
)


print(
    "Expected protocol:",
    EXPECTED_PROTOCOL_SHA256
)

print(
    "Actual protocol  :",
    protocol_sha
)


if protocol_sha != EXPECTED_PROTOCOL_SHA256:

    raise RuntimeError(
        "Measurement protocol identity mismatch."
    )


protocol = json.loads(
    PROTOCOL.read_text(
        encoding="utf-8"
    )
)


pareto = protocol[
    "pareto_protocol"
]

group_a = pareto[
    "GROUP_A_DUPSAFE70"
]

group_b = pareto[
    "GROUP_B_PACKET_IMAGE"
]

dominance_rule = pareto[
    "dominance_rule"
]

operational_refs = pareto[
    "operational_references"
]


expected_group_a = EXPECTED_ORDER[
    :4
]

expected_group_b = EXPECTED_ORDER[
    4:
]


if group_a[
    "eligible_primary_members"
] != expected_group_a:

    raise RuntimeError(
        "Group-A universe mismatch."
    )


if group_b[
    "eligible_primary_members"
] != expected_group_b:

    raise RuntimeError(
        "Group-B universe mismatch."
    )


if group_a[
    "primary_cost_axis"
] != (
    "CPU_1_PHYSICAL_CORE batch1 p95 inference latency"
):

    raise RuntimeError(
        "Group-A cost axis changed."
    )


if group_b[
    "primary_cost_axis"
] != (
    "CPU_1_PHYSICAL_CORE batch1 p95 inference latency"
):

    raise RuntimeError(
        "Group-B cost axis changed."
    )


if group_a[
    "primary_discrimination_axis"
] != (
    "Same-population frozen PR-AUC"
):

    raise RuntimeError(
        "Group-A discrimination axis changed."
    )


if group_b[
    "primary_discrimination_axis"
] != (
    "Same-Friday-population frozen PR-AUC"
):

    raise RuntimeError(
        "Group-B discrimination axis changed."
    )


if group_b[
    "claim_boundary"
] != "DESCRIPTIVE_NON_CONFIRMATORY":

    raise RuntimeError(
        "Group-B claim boundary changed."
    )


if pareto[
    "global_rule"
] != "NO CROSS-GROUP PARETO FRONTIER.":

    raise RuntimeError(
        "Cross-group prohibition changed."
    )


if dominance_rule[
    "A_dominates_B_if"
] != (
    "A PR-AUC >= B PR-AUC and A p95 latency <= B p95 latency, "
    "with at least one strict inequality."
):

    raise RuntimeError(
        "Dominance rule changed."
    )


if dominance_rule[
    "frontier_status"
] != (
    "Descriptive point-estimate frontier; bootstrap uncertainty "
    "reported separately."
):

    raise RuntimeError(
        "Frontier reporting rule changed."
    )


if (
    "excluded from predictive Pareto"
    not in
    operational_refs[
        "FT_BALANCED_SINGLE_RESOURCE_REFERENCE"
    ]
):

    raise RuntimeError(
        "FT resource reference exclusion changed."
    )


print(
    "Group A        : 4 primary points"
)

print(
    "Group B        : 2 primary points"
)

print(
    "Cost axis      : CPU1 B=1 p95 inference latency"
)

print(
    "Dominance rule : verified"
)

print(
    "Cross-group    : FORBIDDEN"
)

print(
    "Group B claim  : DESCRIPTIVE_NON_CONFIRMATORY"
)

print(
    "Dominance now  : NOT COMPUTED"
)


# =============================================================================
# 8. CREATE DURABLE LOCK
# =============================================================================

banner(
    "STAGE26-6C-R :: WRITE DURABLE POINT LOCK"
)


CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


binary64_latency_identities = {
    point[
        "target_id"
    ]:
        float(
            point[
                "p95_cpu1_batch1_inference_latency_ms"
            ]
        ).hex()
    for point in points
}


point_lock = {
    "schema":
        "stage26_6c_resolved_pareto_points_lock_v2",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-6C",

    "status":
        "FROZEN_BEFORE_DOMINANCE",

    "scientific_parent":
        EXPECTED_PARENT,

    "source_stage26_6b": {
        "path":
            str(
                SOURCE_6B
            ),

        "sha256":
            EXPECTED_6B_SHA256,

        "point_set_sha256":
            EXPECTED_POINT_CORE_SHA256,
    },

    "point_count":
        6,

    "point_order":
        EXPECTED_ORDER,

    "points":
        points,

    "point_set_sha256":
        EXPECTED_POINT_CORE_SHA256,

    "cpu1_batch1_p95_binary64_identities":
        binary64_latency_identities,

    "binary64_validation_policy":
        (
            "CPU1/B1 p95 values were verified by exact Python float.hex() "
            "identity against Stage26-6B / Stage26-2 values; no tolerance "
            "or rounding substitution was used."
        ),

    "group_partition": {
        "GROUP_A_DUPSAFE70":
            expected_group_a,

        "GROUP_B_PACKET_IMAGE":
            expected_group_b,
    },

    "source_resolution_policy": {
        "GROUP_A_DUPSAFE70":
            (
                "Frozen PR-AUC values share the exact duplicate-safe "
                "validation population inherited by Stage15/Stage16."
            ),

        "GROUP_B_PACKET_IMAGE":
            (
                "Frozen PR-AUC values share the exact same Friday compact "
                "corpus export order and remain descriptive/non-confirmatory."
            ),

        "cost_axis":
            (
                "CPU_1_PHYSICAL_CORE batch1 p95 isolated inference latency."
            ),
    },

    "excluded_from_primary_point_set": {
        "ENS_LGBM_XGB_EQUAL":
            (
                "Operational reference only; not an additional architecture "
                "family primary point."
            ),

        "FT_BALANCED_SINGLE_RESOURCE_REFERENCE":
            (
                "Resource decomposition only; excluded from predictive Pareto."
            ),
    },

    "scientific_state_at_freeze": {
        "dominance_computed":
            False,

        "frontier_computed":
            False,

        "bootstrap_uncertainty_computed":
            False,

        "cross_group_frontier_computed":
            False,

        "predictive_metric_recomputed":
            False,

        "holdout_reopened":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "pcap_accessed":
            False,

        "release_corpus_accessed":
            False,

        "gpu_used":
            False,
    },
}


rule_lock = {
    "schema":
        "stage26_6c_pareto_rules_lock_v2",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-6C",

    "measurement_protocol_path":
        str(
            PROTOCOL.relative_to(
                REPO
            )
        ),

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "GROUP_A_DUPSAFE70":
        group_a,

    "GROUP_B_PACKET_IMAGE":
        group_b,

    "dominance_rule":
        dominance_rule,

    "global_rule":
        pareto[
            "global_rule"
        ],

    "operational_references":
        operational_refs,

    "execution_guard": {
        "point_estimate_frontier_may_be_computed_only_after_this_lock_is_git_anchored":
            True,

        "bootstrap_uncertainty_is_separate_from_point_estimate_frontier":
            True,

        "cross_group_frontier_prohibited":
            True,

        "post_freeze_metric_substitution_prohibited":
            True,

        "post_freeze_cost_axis_substitution_prohibited":
            True,
    },
}


atomic_json(
    POINT_LOCK_PATH,
    point_lock,
)

atomic_json(
    RULE_LOCK_PATH,
    rule_lock,
)


point_lock_sha = sha256_file(
    POINT_LOCK_PATH
)

rule_lock_sha = sha256_file(
    RULE_LOCK_PATH
)


# =============================================================================
# 9. FREEZE RECEIPT
# =============================================================================

receipt = {
    "schema":
        "stage26_6c_pareto_point_freeze_receipt_v2",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-6C",

    "status":
        "PASS_SIX_PRIMARY_PARETO_POINTS_FROZEN_BEFORE_DOMINANCE",

    "scientific_parent":
        EXPECTED_PARENT,

    "source_6b_sha256":
        EXPECTED_6B_SHA256,

    "point_set_sha256":
        EXPECTED_POINT_CORE_SHA256,

    "point_lock_sha256":
        point_lock_sha,

    "rule_lock_sha256":
        rule_lock_sha,

    "point_count":
        6,

    "group_A_point_count":
        4,

    "group_B_point_count":
        2,

    "exact_binary64_latency_validation":
        True,

    "dominance_computed":
        False,

    "frontier_computed":
        False,

    "bootstrap_uncertainty_computed":
        False,

    "cross_group_frontier_computed":
        False,

    "scientific_boundaries": {
        "PR_AUC_recomputed":
            False,

        "holdout_reopened":
            False,

        "models_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "PCAP_accessed":
            False,

        "Release_corpus_accessed":
            False,

        "GPU_used":
            False,
    },

    "next_permitted_action":
        (
            "After remote Git verification, compute within-group "
            "point-estimate Pareto dominance using only these frozen points."
        ),
}


atomic_json(
    RECEIPT_PATH,
    receipt,
)


receipt_sha = sha256_file(
    RECEIPT_PATH
)


# =============================================================================
# 10. MANIFEST
# =============================================================================

package_files = [
    POINT_LOCK_PATH,
    RULE_LOCK_PATH,
    RECEIPT_PATH,
]


manifest_rows = []


for path in package_files:

    manifest_rows.append(
        {
            "repo_relative_path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest = {
    "schema":
        "stage26_6c_pareto_point_lock_manifest_v2",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        "READY_FOR_GIT_ANCHOR",

    "scientific_parent":
        EXPECTED_PARENT,

    "commit_subject":
        COMMIT_SUBJECT,

    "source_stage26_6b_sha256":
        EXPECTED_6B_SHA256,

    "point_set_sha256":
        EXPECTED_POINT_CORE_SHA256,

    "point_lock_sha256":
        point_lock_sha,

    "rule_lock_sha256":
        rule_lock_sha,

    "receipt_sha256":
        receipt_sha,

    "file_count_excluding_manifest":
        len(
            manifest_rows
        ),

    "files":
        manifest_rows,

    "scientific_state": {
        "point_inputs_frozen":
            True,

        "dominance_computed":
            False,

        "frontier_computed":
            False,

        "bootstrap_uncertainty_computed":
            False,

        "cross_group_frontier_computed":
            False,

        "new_measurement_performed":
            False,

        "GPU_used":
            False,
    },
}


atomic_json(
    MANIFEST_PATH,
    manifest,
)


manifest_sha = sha256_file(
    MANIFEST_PATH
)


print(
    "Point lock:",
    point_lock_sha
)

print(
    "Rule lock :",
    rule_lock_sha
)

print(
    "Receipt   :",
    receipt_sha
)

print(
    "Manifest  :",
    manifest_sha
)


# =============================================================================
# 11. LOCAL BYTE AUDIT
# =============================================================================

banner(
    "STAGE26-6C-R :: LOCAL PACKAGE AUDIT"
)


for row in manifest_rows:

    path = (
        REPO
        /
        row[
            "repo_relative_path"
        ]
    )

    actual_size = int(
        path.stat().st_size
    )

    actual_sha = sha256_file(
        path
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Local Stage26-6C package audit failed."
        )


# =============================================================================
# 12. GIT CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-6C-R :: GIT CHANGE AUDIT"
)


repo_status = git(
    "status",
    "--porcelain",
)


print(
    repo_status
)


if not repo_status:

    raise RuntimeError(
        "Expected uncommitted Stage26-6C package."
    )


unexpected = []


for line in repo_status.splitlines():

    relpath = line[
        3:
    ]


    if not relpath.startswith(
        str(
            CHECKPOINT_REL
        )
        +
        "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository changes:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 13. GIT IDENTITY
# =============================================================================

author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_PARENT,
)

author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_PARENT,
)


if (
    not author_name.strip()
    or
    "@"
    not in
    author_email
):

    raise RuntimeError(
        "Could not recover valid Git identity."
    )


git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


print(
    "\nGit author:"
)

print(
    " ",
    author_name,
    "<" + author_email + ">"
)


# =============================================================================
# 14. COMMIT
# =============================================================================

banner(
    "STAGE26-6C-R :: COMMIT"
)


git(
    "add",
    str(
        CHECKPOINT_REL
    ),
)


print(
    git(
        "diff",
        "--cached",
        "--name-status",
    )
)


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-6C commit parent mismatch."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Stage26-6C commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository not clean after 6C commit."
    )


# =============================================================================
# 15. PUSH
# =============================================================================

banner(
    "STAGE26-6C-R :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    push_result = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
        text=True,
    )


    print(
        push_result.stdout.strip()
    )


github_token = None


# =============================================================================
# 16. REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-6C-R :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "origin/main did not advance to Stage26-6C."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote Stage26-6C parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote Stage26-6C subject mismatch."
    )


# =============================================================================
# 17. REMOTE BYTE AUDIT
# =============================================================================

banner(
    "STAGE26-6C-R :: REMOTE BYTE VERIFICATION"
)


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    str(
        MANIFEST_PATH.relative_to(
            REPO
        )
    ),
)


remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "Remote manifest SHA256:"
)

print(
    " ",
    remote_manifest_sha
)


if remote_manifest_sha != manifest_sha:

    raise RuntimeError(
        "Remote Stage26-6C manifest mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


for row in remote_manifest[
    "files"
]:

    data = git_blob_bytes(
        "origin/main",
        row[
            "repo_relative_path"
        ],
    )

    actual_size = len(
        data
    )

    actual_sha = sha256_bytes(
        data
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Remote Stage26-6C byte verification failed."
        )


# =============================================================================
# 18. REMOTE SCIENTIFIC AUDIT
# =============================================================================

banner(
    "STAGE26-6C-R :: REMOTE SCIENTIFIC AUDIT"
)


remote_lock = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            POINT_LOCK_PATH.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_rules = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            RULE_LOCK_PATH.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_receipt = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            RECEIPT_PATH.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)


remote_checks = {
    "six points":
        (
            len(
                remote_lock[
                    "points"
                ]
            )
            ==
            6
        ),

    "point SHA":
        (
            remote_lock[
                "point_set_sha256"
            ]
            ==
            EXPECTED_POINT_CORE_SHA256
        ),

    "binary64 validation":
        (
            remote_receipt[
                "exact_binary64_latency_validation"
            ]
            is True
        ),

    "frozen before dominance":
        (
            remote_lock[
                "status"
            ]
            ==
            "FROZEN_BEFORE_DOMINANCE"
        ),

    "dominance false":
        (
            remote_receipt[
                "dominance_computed"
            ]
            is False
        ),

    "frontier false":
        (
            remote_receipt[
                "frontier_computed"
            ]
            is False
        ),

    "bootstrap false":
        (
            remote_receipt[
                "bootstrap_uncertainty_computed"
            ]
            is False
        ),

    "cross-group false":
        (
            remote_receipt[
                "cross_group_frontier_computed"
            ]
            is False
        ),

    "Group A count":
        (
            remote_receipt[
                "group_A_point_count"
            ]
            ==
            4
        ),

    "Group B count":
        (
            remote_receipt[
                "group_B_point_count"
            ]
            ==
            2
        ),

    "global rule":
        (
            remote_rules[
                "global_rule"
            ]
            ==
            "NO CROSS-GROUP PARETO FRONTIER."
        ),

    "GPU false":
        (
            remote_receipt[
                "scientific_boundaries"
            ][
                "GPU_used"
            ]
            is False
        ),
}


for name, passed in remote_checks.items():

    print(
        f"{name:34s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    remote_checks.values()
):

    raise RuntimeError(
        "Remote Stage26-6C scientific audit failed."
    )


# Exact remote binary64 verification.
for remote_point in remote_lock[
    "points"
]:

    target = remote_point[
        "target_id"
    ]

    expected = EXPECTED_POINTS[
        target
    ]

    actual_hex = float(
        remote_point[
            "p95_cpu1_batch1_inference_latency_ms"
        ]
    ).hex()


    if actual_hex != expected[
        "latency_hex"
    ]:

        raise RuntimeError(
            f"Remote binary64 latency mismatch: {target}"
        )


    if float(
        remote_point[
            "pr_auc"
        ]
    ) != float(
        expected[
            "pr_auc"
        ]
    ):

        raise RuntimeError(
            f"Remote PR-AUC mismatch: {target}"
        )


    if remote_point[
        "cost_source"
    ][
        "condition_id"
    ] != expected[
        "condition_id"
    ]:

        raise RuntimeError(
            f"Remote condition mismatch: {target}"
        )


    print(
        f"PASS {target:42s} "
        f"PR={float(remote_point['pr_auc']):.15f} "
        f"p95={repr(float(remote_point['p95_cpu1_batch1_inference_latency_ms']))} "
        f"hex={actual_hex}"
    )


# =============================================================================
# 19. FINAL CLOSURE
# =============================================================================

banner(
    "STAGE26-6C EXACT PARETO INPUT FREEZE COMPLETE"
)


final_status = git(
    "status",
    "--porcelain",
)


print(
    "NEW DURABLE COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nPARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nFROZEN POINT CORE:"
)

print(
    " ",
    EXPECTED_POINT_CORE_SHA256
)


print(
    "\nPACKAGE HASHES:"
)

print(
    "  point lock:",
    point_lock_sha
)

print(
    "  rule lock :",
    rule_lock_sha
)

print(
    "  receipt   :",
    receipt_sha
)

print(
    "  manifest  :",
    manifest_sha
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  six primary points frozen       : YES"
)

print(
    "  exact binary64 p95 identity      : YES"
)

print(
    "  dominance computed              : NO"
)

print(
    "  frontier computed               : NO"
)

print(
    "  bootstrap uncertainty computed  : NO"
)

print(
    "  cross-group frontier            : NO"
)

print(
    "  metric recomputation            : NO"
)

print(
    "  new inference                   : NO"
)

print(
    "  new timing                      : NO"
)

print(
    "  holdout reopened                : NO"
)

print(
    "  GPU                             : NO"
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  commit              : PASS"
)

print(
    "  parent              : PASS"
)

print(
    "  package bytes       : PASS"
)

print(
    "  exact six points    : PASS"
)

print(
    "  binary64 identities : PASS"
)

print(
    "  scientific content  : PASS"
)


print(
    "\nRepo clean:",
    final_status == ""
)


if final_status:

    raise RuntimeError(
        "Repository not clean after Stage26-6C recovery."
    )


print(
    "\nNEXT:"
)

print(
    "  Stage26-6D may now compute the frozen within-group"
)

print(
    "  point-estimate Pareto dominance/frontiers."
)

print(
    "  GROUP A and GROUP B remain strictly separate."
)

print(
    "  Group B remains descriptive/non-confirmatory."
)

print(
    "  Bootstrap uncertainty remains separate."
)

print(
    "  No GPU yet."
)


STAGE26-6C-R :: DURABLE STATE
Expected parent: 92495eac2c9202973ff4d889e3c12669c06e9ef1
Local HEAD     : 92495eac2c9202973ff4d889e3c12669c06e9ef1
origin/main    : 92495eac2c9202973ff4d889e3c12669c06e9ef1
Repo clean     : True
6C checkpoint exists: False

[PASS] Original 6C failed before any checkpoint write.

STAGE26-6C-R :: STAGE26-6B IDENTITY
Expected 6B SHA256: 715c05709f865477aacc9546377f45f9484eefc9a55559506e3d42fac26a44a7
Actual   6B SHA256: 715c05709f865477aacc9546377f45f9484eefc9a55559506e3d42fac26a44a7
Point set SHA : PASS
Dominance     : NOT COMPUTED
Frontier      : NOT COMPUTED
Bootstrap     : NOT COMPUTED

STAGE26-6C-R :: EXACT SIX-POINT BINARY GATE

STAGE16_XGBOOST_TUNED
  group               : PASS
  PR-AUC exact        : PASS
  p95 binary64 exact  : PASS
  condition ID        : PASS
  CPU1                : PASS
  affinity            : PASS
  threads             : PASS
  batch               : PASS
  timed runs          : PASS
  condition PASS      : PASS
  PR-AUC repr   

In [16]:
# =============================================================================
# STAGE26-6D
# COMPUTE FROZEN WITHIN-GROUP POINT-ESTIMATE PARETO FRONTIERS
# COMMIT + PUSH + REMOTE VERIFY
#
# DURABLE SCIENTIFIC PARENT:
#   36a1767c81ccc44818c543e82c2b8da95be39560
#
# INPUTS:
#   Stage26-6C exact six-point lock
#   Stage26-6C frozen Pareto-rule lock
#
# FROZEN DOMINANCE:
#
#   A dominates B iff:
#
#       A PR-AUC >= B PR-AUC
#       AND
#       A p95 latency <= B p95 latency
#       AND
#       at least one inequality is strict
#
# GROUPS MUST REMAIN SEPARATE:
#
#   GROUP_A_DUPSAFE70
#       XGBoost
#       LightGBM
#       CatBoost
#       FT 5-checkpoint ensemble
#
#   GROUP_B_PACKET_IMAGE
#       CNN
#       ViT
#
# GROUP-B CLAIM:
#   DESCRIPTIVE_NON_CONFIRMATORY
#
# THIS CELL:
#   - verifies exact Stage26-6C lock identity;
#   - computes every ordered within-group dominance relation;
#   - computes nondominated point-estimate frontier membership;
#   - performs NO cross-group comparison;
#   - performs NO bootstrap uncertainty;
#   - writes pairwise dominance CSV + frontier JSON/CSV + receipt/manifest;
#   - commits, pushes, and remotely verifies.
#
# NO:
#   - PR-AUC recomputation
#   - holdout reopening
#   - model loading
#   - inference
#   - timing
#   - bootstrap
#   - PCAP
#   - Release corpus
#   - GPU
# =============================================================================

from __future__ import annotations

import csv
import json
import os
import stat
import hashlib
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "36a1767c81ccc44818c543e82c2b8da95be39560"
)

COMMIT_SUBJECT = (
    "stage26: anchor CPU Pareto point-estimate frontiers"
)


LOCK_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_6c_pareto_point_lock"
)

POINT_LOCK = (
    LOCK_DIR
    / "stage26_6c_resolved_pareto_points_lock.json"
)

RULE_LOCK = (
    LOCK_DIR
    / "stage26_6c_pareto_rules_lock.json"
)

FREEZE_RECEIPT = (
    LOCK_DIR
    / "stage26_6c_pareto_point_freeze_receipt.json"
)

LOCK_MANIFEST = (
    LOCK_DIR
    / "stage26_6c_pareto_point_lock_manifest.json"
)


EXPECTED_POINT_LOCK_SHA256 = (
    "3b4739d9cec40c0dc1a21c80cb3cc4682cab08bef671f760de4697a7ad8d6ddf"
)

EXPECTED_RULE_LOCK_SHA256 = (
    "aff5821c4e870bbd7326d93563c6e952cf34aafae06fa483e9ec6f348409a1d2"
)

EXPECTED_FREEZE_RECEIPT_SHA256 = (
    "f86db80ce6a611963671603fb3a27b260649fd6af36610cb5d2fbcd920da90b8"
)

EXPECTED_LOCK_MANIFEST_SHA256 = (
    "ddb1320491a54e4d6c5d461fb61c4854bfbf08e8f9585be2afff4ce8c8721479"
)

EXPECTED_POINT_SET_SHA256 = (
    "5fb0e952d6560b6cd0f84c49048c05d26ff3c89c52edd61ea91855d374c10a76"
)


GROUP_A = (
    "GROUP_A_DUPSAFE70"
)

GROUP_B = (
    "GROUP_B_PACKET_IMAGE"
)


EXPECTED_GROUP_MEMBERS = {
    GROUP_A: [
        "STAGE16_XGBOOST_TUNED",
        "STAGE16_LIGHTGBM_TUNED",
        "STAGE16_CATBOOST_TUNED",
        "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    ],

    GROUP_B: [
        "STAGE20_MASKED_CNN_V1",
        "STAGE21_MASKED_VIT_V1",
    ],
}


# These are deterministic expectations from the already-frozen inputs/rule.
# They are used ONLY as an implementation audit after computation.
EXPECTED_FRONTIER = {
    GROUP_A: [
        "STAGE16_XGBOOST_TUNED",
        "STAGE16_LIGHTGBM_TUNED",
        "STAGE16_CATBOOST_TUNED",
    ],

    GROUP_B: [
        "STAGE21_MASKED_VIT_V1",
    ],
}


EXPECTED_DOMINATORS = {
    "STAGE16_XGBOOST_TUNED": [],
    "STAGE16_LIGHTGBM_TUNED": [],
    "STAGE16_CATBOOST_TUNED": [],

    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING": [
        "STAGE16_XGBOOST_TUNED",
        "STAGE16_LIGHTGBM_TUNED",
        "STAGE16_CATBOOST_TUNED",
    ],

    "STAGE20_MASKED_CNN_V1": [
        "STAGE21_MASKED_VIT_V1",
    ],

    "STAGE21_MASKED_VIT_V1": [],
}


# =============================================================================
# 1. OUTPUTS
# =============================================================================

CHECKPOINT_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_6d_cpu_pareto_point_estimate"
)

CHECKPOINT_DIR = (
    REPO
    / CHECKPOINT_REL
)

PAIRWISE_CSV = (
    CHECKPOINT_DIR
    / "stage26_6d_within_group_pairwise_dominance.csv"
)

FRONTIER_JSON = (
    CHECKPOINT_DIR
    / "stage26_6d_point_estimate_frontiers.json"
)

FRONTIER_CSV = (
    CHECKPOINT_DIR
    / "stage26_6d_point_estimate_frontiers.csv"
)

RECEIPT_PATH = (
    CHECKPOINT_DIR
    / "stage26_6d_pareto_point_estimate_receipt.json"
)

MANIFEST_PATH = (
    CHECKPOINT_DIR
    / "stage26_6d_pareto_point_estimate_manifest.json"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            + " ".join(
                map(
                    str,
                    cmd,
                )
            )
            + "\n\n"
            + output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_json(
    path,
    obj,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        text=False,
    )

    return bytes(
        p.stdout
    )


def dominates(
    point_a,
    point_b,
):

    pr_a = float(
        point_a[
            "pr_auc"
        ]
    )

    pr_b = float(
        point_b[
            "pr_auc"
        ]
    )

    latency_a = float(
        point_a[
            "p95_cpu1_batch1_inference_latency_ms"
        ]
    )

    latency_b = float(
        point_b[
            "p95_cpu1_batch1_inference_latency_ms"
        ]
    )

    no_worse_discrimination = (
        pr_a >= pr_b
    )

    no_worse_latency = (
        latency_a <= latency_b
    )

    at_least_one_strict = (
        pr_a > pr_b
        or
        latency_a < latency_b
    )

    return (
        no_worse_discrimination
        and
        no_worse_latency
        and
        at_least_one_strict
    )


# =============================================================================
# 3. DURABLE GIT GATE
# =============================================================================

banner(
    "STAGE26-6D :: DURABLE SCIENTIFIC PARENT"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-6D parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Pareto computation."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if CHECKPOINT_DIR.exists():

    raise RuntimeError(
        "Stage26-6D checkpoint already exists."
    )


# =============================================================================
# 4. STAGE26-6C LOCK IDENTITY
# =============================================================================

banner(
    "STAGE26-6D :: FROZEN INPUT IDENTITY"
)


identity_checks = [
    (
        "point lock",
        POINT_LOCK,
        EXPECTED_POINT_LOCK_SHA256,
    ),

    (
        "rule lock",
        RULE_LOCK,
        EXPECTED_RULE_LOCK_SHA256,
    ),

    (
        "freeze receipt",
        FREEZE_RECEIPT,
        EXPECTED_FREEZE_RECEIPT_SHA256,
    ),

    (
        "lock manifest",
        LOCK_MANIFEST,
        EXPECTED_LOCK_MANIFEST_SHA256,
    ),
]


for label, path, expected in identity_checks:

    if not path.is_file():

        raise FileNotFoundError(
            path
        )


    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:22s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Stage26-6C identity mismatch: {label}"
        )


# =============================================================================
# 5. LOAD EXACT FROZEN INPUTS
# =============================================================================

banner(
    "STAGE26-6D :: FROZEN SCIENTIFIC INPUT GATE"
)


point_lock = json.loads(
    POINT_LOCK.read_text(
        encoding="utf-8"
    )
)

rule_lock = json.loads(
    RULE_LOCK.read_text(
        encoding="utf-8"
    )
)

freeze_receipt = json.loads(
    FREEZE_RECEIPT.read_text(
        encoding="utf-8"
    )
)


if point_lock[
    "status"
] != "FROZEN_BEFORE_DOMINANCE":

    raise RuntimeError(
        "Point lock was not frozen before dominance."
    )


if point_lock[
    "point_set_sha256"
] != EXPECTED_POINT_SET_SHA256:

    raise RuntimeError(
        "Frozen point-set SHA mismatch."
    )


if freeze_receipt[
    "dominance_computed"
] is not False:

    raise RuntimeError(
        "Dominance was already computed before Stage26-6D."
    )


if freeze_receipt[
    "frontier_computed"
] is not False:

    raise RuntimeError(
        "Frontier was already computed before Stage26-6D."
    )


if freeze_receipt[
    "bootstrap_uncertainty_computed"
] is not False:

    raise RuntimeError(
        "Bootstrap was already computed before Stage26-6D."
    )


if rule_lock[
    "global_rule"
] != "NO CROSS-GROUP PARETO FRONTIER.":

    raise RuntimeError(
        "Frozen cross-group prohibition changed."
    )


if rule_lock[
    "dominance_rule"
][
    "A_dominates_B_if"
] != (
    "A PR-AUC >= B PR-AUC and A p95 latency <= B p95 latency, "
    "with at least one strict inequality."
):

    raise RuntimeError(
        "Frozen dominance rule changed."
    )


if rule_lock[
    "GROUP_B_PACKET_IMAGE"
][
    "claim_boundary"
] != "DESCRIPTIVE_NON_CONFIRMATORY":

    raise RuntimeError(
        "Group-B claim boundary changed."
    )


points = point_lock[
    "points"
]


if len(
    points
) != 6:

    raise RuntimeError(
        "Frozen primary point count is not six."
    )


points_by_id = {
    point[
        "target_id"
    ]:
        point
    for point in points
}


if set(
    points_by_id
) != set(
    EXPECTED_GROUP_MEMBERS[
        GROUP_A
    ]
    +
    EXPECTED_GROUP_MEMBERS[
        GROUP_B
    ]
):

    raise RuntimeError(
        "Frozen point universe changed."
    )


print(
    "Frozen point count : 6"
)

print(
    "Point-core SHA     :",
    EXPECTED_POINT_SET_SHA256
)

print(
    "Group A            : 4"
)

print(
    "Group B            : 2"
)

print(
    "Cross-group        : PROHIBITED"
)

print(
    "Bootstrap          : NOT PERFORMED"
)


# =============================================================================
# 6. PARTITION — EXACTLY TWO GROUPS
# =============================================================================

grouped = {
    GROUP_A: [],
    GROUP_B: [],
}


for point in points:

    group = point[
        "comparison_group"
    ]


    if group not in grouped:

        raise RuntimeError(
            f"Unexpected comparison group: {group}"
        )


    grouped[
        group
    ].append(
        point
    )


for group, expected_members in EXPECTED_GROUP_MEMBERS.items():

    actual_members = [
        point[
            "target_id"
        ]
        for point in grouped[
            group
        ]
    ]


    if actual_members != expected_members:

        raise RuntimeError(
            f"{group} membership/order changed."
        )


# =============================================================================
# 7. COMPUTE WITHIN-GROUP ORDERED PAIRWISE DOMINANCE
# =============================================================================

banner(
    "STAGE26-6D :: COMPUTE WITHIN-GROUP DOMINANCE"
)


pairwise_rows = []

dominators = {
    point[
        "target_id"
    ]:
        []
    for point in points
}


for group in [
    GROUP_A,
    GROUP_B,
]:

    group_points = grouped[
        group
    ]


    print(
        "\n",
        group,
        sep="",
    )


    for point_a in group_points:

        for point_b in group_points:

            if point_a[
                "target_id"
            ] == point_b[
                "target_id"
            ]:

                continue


            result = dominates(
                point_a,
                point_b,
            )


            pr_a = float(
                point_a[
                    "pr_auc"
                ]
            )

            pr_b = float(
                point_b[
                    "pr_auc"
                ]
            )

            latency_a = float(
                point_a[
                    "p95_cpu1_batch1_inference_latency_ms"
                ]
            )

            latency_b = float(
                point_b[
                    "p95_cpu1_batch1_inference_latency_ms"
                ]
            )


            pairwise_rows.append(
                {
                    "comparison_group":
                        group,

                    "model_A":
                        point_a[
                            "target_id"
                        ],

                    "model_B":
                        point_b[
                            "target_id"
                        ],

                    "A_pr_auc":
                        pr_a,

                    "B_pr_auc":
                        pr_b,

                    "A_p95_cpu1_batch1_latency_ms":
                        latency_a,

                    "B_p95_cpu1_batch1_latency_ms":
                        latency_b,

                    "A_pr_auc_ge_B":
                        pr_a >= pr_b,

                    "A_latency_le_B":
                        latency_a <= latency_b,

                    "at_least_one_strict":
                        (
                            pr_a > pr_b
                            or
                            latency_a < latency_b
                        ),

                    "A_dominates_B":
                        result,
                }
            )


            if result:

                dominators[
                    point_b[
                        "target_id"
                    ]
                ].append(
                    point_a[
                        "target_id"
                    ]
                )


                print(
                    "  ",
                    point_a[
                        "target_id"
                    ],
                    "DOMINATES",
                    point_b[
                        "target_id"
                    ],
                )


expected_pair_count = (
    4 * 3
    +
    2 * 1
)


if len(
    pairwise_rows
) != expected_pair_count:

    raise RuntimeError(
        "Unexpected pairwise comparison count."
    )


print(
    "\nOrdered comparisons:",
    len(
        pairwise_rows
    )
)

print(
    "Expected           :",
    expected_pair_count
)


# =============================================================================
# 8. COMPUTE FRONTIER MEMBERSHIP
# =============================================================================

banner(
    "STAGE26-6D :: COMPUTE POINT-ESTIMATE FRONTIERS"
)


frontier_by_group = {}


for group in [
    GROUP_A,
    GROUP_B,
]:

    frontier = [
        point[
            "target_id"
        ]
        for point in grouped[
            group
        ]
        if len(
            dominators[
                point[
                    "target_id"
                ]
            ]
        ) == 0
    ]


    frontier_by_group[
        group
    ] = frontier


    print(
        "\n",
        group,
        sep="",
    )

    print(
        "  frontier:",
        frontier
    )


    for point in grouped[
        group
    ]:

        target = point[
            "target_id"
        ]


        print(
            f"  {target:42s} "
            f"frontier={'YES' if target in frontier else 'NO ':3s} "
            f"dominators={dominators[target]}"
        )


# =============================================================================
# 9. INDEPENDENT IMPLEMENTATION AUDIT
# =============================================================================

banner(
    "STAGE26-6D :: DETERMINISTIC RESULT AUDIT"
)


for target, expected in EXPECTED_DOMINATORS.items():

    actual = sorted(
        dominators[
            target
        ]
    )

    expected_sorted = sorted(
        expected
    )


    passed = (
        actual
        ==
        expected_sorted
    )


    print(
        f"{target:42s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"dominators={actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Unexpected dominator set for {target}."
        )


for group, expected_frontier in EXPECTED_FRONTIER.items():

    actual = frontier_by_group[
        group
    ]


    passed = (
        actual
        ==
        expected_frontier
    )


    print(
        f"{group:28s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"frontier={actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Unexpected point-estimate frontier for {group}."
        )


# Ensure zero cross-group rows were ever created.
for row in pairwise_rows:

    a_group = points_by_id[
        row[
            "model_A"
        ]
    ][
        "comparison_group"
    ]

    b_group = points_by_id[
        row[
            "model_B"
        ]
    ][
        "comparison_group"
    ]


    if a_group != b_group:

        raise RuntimeError(
            "Cross-group dominance comparison detected."
        )


print(
    "\nCross-group dominance comparisons: 0"
)


# =============================================================================
# 10. CREATE DURABLE OUTPUT DIRECTORY
# =============================================================================

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# =============================================================================
# 11. WRITE PAIRWISE CSV
# =============================================================================

pairwise_fields = [
    "comparison_group",
    "model_A",
    "model_B",
    "A_pr_auc",
    "B_pr_auc",
    "A_p95_cpu1_batch1_latency_ms",
    "B_p95_cpu1_batch1_latency_ms",
    "A_pr_auc_ge_B",
    "A_latency_le_B",
    "at_least_one_strict",
    "A_dominates_B",
]


with PAIRWISE_CSV.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=pairwise_fields,
    )

    writer.writeheader()

    writer.writerows(
        pairwise_rows
    )


pairwise_sha = sha256_file(
    PAIRWISE_CSV
)


# =============================================================================
# 12. WRITE FRONTIER JSON
# =============================================================================

frontier_records = []


for group in [
    GROUP_A,
    GROUP_B,
]:

    for point in grouped[
        group
    ]:

        target = point[
            "target_id"
        ]


        frontier_records.append(
            {
                "target_id":
                    target,

                "comparison_group":
                    group,

                "pr_auc":
                    float(
                        point[
                            "pr_auc"
                        ]
                    ),

                "p95_cpu1_batch1_inference_latency_ms":
                    float(
                        point[
                            "p95_cpu1_batch1_inference_latency_ms"
                        ]
                    ),

                "frontier_member":
                    (
                        target
                        in
                        frontier_by_group[
                            group
                        ]
                    ),

                "dominator_count":
                    len(
                        dominators[
                            target
                        ]
                    ),

                "dominators":
                    dominators[
                        target
                    ],
            }
        )


frontier_payload = {
    "schema":
        "stage26_6d_point_estimate_frontiers_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-6D",

    "status":
        "PASS_WITHIN_GROUP_POINT_ESTIMATE_FRONTIERS",

    "scientific_parent":
        EXPECTED_PARENT,

    "input_point_lock_sha256":
        EXPECTED_POINT_LOCK_SHA256,

    "input_rule_lock_sha256":
        EXPECTED_RULE_LOCK_SHA256,

    "input_point_set_sha256":
        EXPECTED_POINT_SET_SHA256,

    "dominance_rule":
        rule_lock[
            "dominance_rule"
        ][
            "A_dominates_B_if"
        ],

    "frontier_status":
        rule_lock[
            "dominance_rule"
        ][
            "frontier_status"
        ],

    "global_rule":
        rule_lock[
            "global_rule"
        ],

    "groups": {
        GROUP_A: {
            "claim_boundary":
                "PRIMARY_WITHIN_GROUP_POINT_ESTIMATE",

            "member_count":
                4,

            "frontier_member_count":
                len(
                    frontier_by_group[
                        GROUP_A
                    ]
                ),

            "frontier_members":
                frontier_by_group[
                    GROUP_A
                ],
        },

        GROUP_B: {
            "claim_boundary":
                "DESCRIPTIVE_NON_CONFIRMATORY",

            "member_count":
                2,

            "frontier_member_count":
                len(
                    frontier_by_group[
                        GROUP_B
                    ]
                ),

            "frontier_members":
                frontier_by_group[
                    GROUP_B
                ],
        },
    },

    "points":
        frontier_records,

    "pairwise_ordered_comparison_count":
        len(
            pairwise_rows
        ),

    "cross_group_comparison_count":
        0,

    "bootstrap_uncertainty": {
        "computed":
            False,

        "status":
            "SEPARATE_STAGE_NOT_YET_EXECUTED",
    },

    "scientific_boundaries": {
        "predictive_metric_recomputed":
            False,

        "holdout_reopened":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "cross_group_frontier_computed":
            False,

        "operational_reference_added_as_primary":
            False,

        "ft_single_resource_reference_added":
            False,

        "PCAP_accessed":
            False,

        "Release_corpus_accessed":
            False,

        "GPU_used":
            False,
    },
}


atomic_json(
    FRONTIER_JSON,
    frontier_payload,
)


frontier_json_sha = sha256_file(
    FRONTIER_JSON
)


# =============================================================================
# 13. WRITE FRONTIER CSV
# =============================================================================

frontier_fields = [
    "comparison_group",
    "target_id",
    "pr_auc",
    "p95_cpu1_batch1_inference_latency_ms",
    "frontier_member",
    "dominator_count",
    "dominators",
    "claim_boundary",
]


with FRONTIER_CSV.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=frontier_fields,
    )

    writer.writeheader()


    for row in frontier_records:

        group = row[
            "comparison_group"
        ]


        writer.writerow(
            {
                "comparison_group":
                    group,

                "target_id":
                    row[
                        "target_id"
                    ],

                "pr_auc":
                    row[
                        "pr_auc"
                    ],

                "p95_cpu1_batch1_inference_latency_ms":
                    row[
                        "p95_cpu1_batch1_inference_latency_ms"
                    ],

                "frontier_member":
                    row[
                        "frontier_member"
                    ],

                "dominator_count":
                    row[
                        "dominator_count"
                    ],

                "dominators":
                    ";".join(
                        row[
                            "dominators"
                        ]
                    ),

                "claim_boundary":
                    (
                        "DESCRIPTIVE_NON_CONFIRMATORY"
                        if group == GROUP_B
                        else
                        "PRIMARY_WITHIN_GROUP_POINT_ESTIMATE"
                    ),
            }
        )


frontier_csv_sha = sha256_file(
    FRONTIER_CSV
)


# =============================================================================
# 14. RECEIPT
# =============================================================================

receipt = {
    "schema":
        "stage26_6d_pareto_point_estimate_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-6D",

    "status":
        "PASS_FROZEN_WITHIN_GROUP_POINT_ESTIMATE_PARETO",

    "scientific_parent":
        EXPECTED_PARENT,

    "frozen_inputs": {
        "point_lock_sha256":
            EXPECTED_POINT_LOCK_SHA256,

        "rule_lock_sha256":
            EXPECTED_RULE_LOCK_SHA256,

        "freeze_receipt_sha256":
            EXPECTED_FREEZE_RECEIPT_SHA256,

        "lock_manifest_sha256":
            EXPECTED_LOCK_MANIFEST_SHA256,

        "point_set_sha256":
            EXPECTED_POINT_SET_SHA256,
    },

    "results": {
        GROUP_A: {
            "frontier_members":
                frontier_by_group[
                    GROUP_A
                ],

            "dominated_members": [
                target
                for target in EXPECTED_GROUP_MEMBERS[
                    GROUP_A
                ]
                if target not in frontier_by_group[
                    GROUP_A
                ]
            ],
        },

        GROUP_B: {
            "claim_boundary":
                "DESCRIPTIVE_NON_CONFIRMATORY",

            "frontier_members":
                frontier_by_group[
                    GROUP_B
                ],

            "dominated_members": [
                target
                for target in EXPECTED_GROUP_MEMBERS[
                    GROUP_B
                ]
                if target not in frontier_by_group[
                    GROUP_B
                ]
            ],
        },
    },

    "ordered_within_group_comparisons":
        len(
            pairwise_rows
        ),

    "cross_group_comparisons":
        0,

    "output_hashes": {
        "pairwise_csv_sha256":
            pairwise_sha,

        "frontier_json_sha256":
            frontier_json_sha,

        "frontier_csv_sha256":
            frontier_csv_sha,
    },

    "bootstrap_uncertainty_computed":
        False,

    "scientific_state": {
        "point_estimate_dominance_computed":
            True,

        "point_estimate_frontiers_computed":
            True,

        "cross_group_frontier_computed":
            False,

        "predictive_metric_recomputed":
            False,

        "holdout_reopened":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "PCAP_accessed":
            False,

        "Release_corpus_accessed":
            False,

        "GPU_used":
            False,
    },

    "next":
        (
            "Audit frozen bootstrap-uncertainty availability and semantics "
            "separately; do not alter the point-estimate frontier."
        ),
}


atomic_json(
    RECEIPT_PATH,
    receipt,
)


receipt_sha = sha256_file(
    RECEIPT_PATH
)


# =============================================================================
# 15. MANIFEST
# =============================================================================

package_files = [
    PAIRWISE_CSV,
    FRONTIER_JSON,
    FRONTIER_CSV,
    RECEIPT_PATH,
]


manifest_rows = []


for path in package_files:

    manifest_rows.append(
        {
            "repo_relative_path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest = {
    "schema":
        "stage26_6d_pareto_point_estimate_manifest_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        "READY_FOR_GIT_ANCHOR",

    "scientific_parent":
        EXPECTED_PARENT,

    "commit_subject":
        COMMIT_SUBJECT,

    "input_point_lock_sha256":
        EXPECTED_POINT_LOCK_SHA256,

    "input_rule_lock_sha256":
        EXPECTED_RULE_LOCK_SHA256,

    "input_point_set_sha256":
        EXPECTED_POINT_SET_SHA256,

    "pairwise_csv_sha256":
        pairwise_sha,

    "frontier_json_sha256":
        frontier_json_sha,

    "frontier_csv_sha256":
        frontier_csv_sha,

    "receipt_sha256":
        receipt_sha,

    "file_count_excluding_manifest":
        len(
            manifest_rows
        ),

    "files":
        manifest_rows,

    "scientific_state": {
        "point_estimate_dominance_computed":
            True,

        "point_estimate_frontiers_computed":
            True,

        "bootstrap_uncertainty_computed":
            False,

        "cross_group_frontier_computed":
            False,

        "new_measurement_performed":
            False,

        "GPU_used":
            False,
    },
}


atomic_json(
    MANIFEST_PATH,
    manifest,
)


manifest_sha = sha256_file(
    MANIFEST_PATH
)


# =============================================================================
# 16. PRINT RESULTS
# =============================================================================

banner(
    "STAGE26-6D :: POINT-ESTIMATE RESULTS"
)


print(
    "GROUP A — PRIMARY WITHIN-GROUP FRONTIER"
)

for target in frontier_by_group[
    GROUP_A
]:

    point = points_by_id[
        target
    ]


    print(
        f"  {target:42s} "
        f"PR-AUC={float(point['pr_auc']):.15f} "
        f"p95={float(point['p95_cpu1_batch1_inference_latency_ms']):.12f} ms"
    )


print(
    "\nGROUP A — DOMINATED"
)

for target in EXPECTED_GROUP_MEMBERS[
    GROUP_A
]:

    if target in frontier_by_group[
        GROUP_A
    ]:

        continue


    print(
        f"  {target}"
    )

    print(
        "    dominated by:",
        dominators[
            target
        ]
    )


print(
    "\nGROUP B — DESCRIPTIVE/NON-CONFIRMATORY FRONTIER"
)

for target in frontier_by_group[
    GROUP_B
]:

    point = points_by_id[
        target
    ]


    print(
        f"  {target:42s} "
        f"PR-AUC={float(point['pr_auc']):.15f} "
        f"p95={float(point['p95_cpu1_batch1_inference_latency_ms']):.12f} ms"
    )


print(
    "\nGROUP B — DOMINATED"
)

for target in EXPECTED_GROUP_MEMBERS[
    GROUP_B
]:

    if target in frontier_by_group[
        GROUP_B
    ]:

        continue


    print(
        f"  {target}"
    )

    print(
        "    dominated by:",
        dominators[
            target
        ]
    )


print(
    "\nCross-group comparisons:",
    0
)

print(
    "Bootstrap uncertainty : NOT COMPUTED"
)


print(
    "\nOUTPUT HASHES:"
)

print(
    "  pairwise CSV :",
    pairwise_sha
)

print(
    "  frontier JSON:",
    frontier_json_sha
)

print(
    "  frontier CSV :",
    frontier_csv_sha
)

print(
    "  receipt      :",
    receipt_sha
)

print(
    "  manifest     :",
    manifest_sha
)


# =============================================================================
# 17. LOCAL PACKAGE AUDIT
# =============================================================================

banner(
    "STAGE26-6D :: LOCAL PACKAGE AUDIT"
)


for row in manifest_rows:

    path = (
        REPO
        / row[
            "repo_relative_path"
        ]
    )

    size = int(
        path.stat().st_size
    )

    digest = sha256_file(
        path
    )

    passed = (
        size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        digest
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{size:10,d} B "
        f"{digest} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Local Stage26-6D package audit failed."
        )


# =============================================================================
# 18. GIT CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-6D :: GIT CHANGE AUDIT"
)


repo_status = git(
    "status",
    "--porcelain",
)


print(
    repo_status
)


if not repo_status:

    raise RuntimeError(
        "Expected uncommitted Stage26-6D package."
    )


unexpected = []


for line in repo_status.splitlines():

    relpath = line[
        3:
    ]


    if not relpath.startswith(
        str(
            CHECKPOINT_REL
        )
        + "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository changes:\n"
        + "\n".join(
            unexpected
        )
    )


# =============================================================================
# 19. GIT IDENTITY
# =============================================================================

author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_PARENT,
)

author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_PARENT,
)


if (
    not author_name.strip()
    or
    "@"
    not in
    author_email
):

    raise RuntimeError(
        "Could not recover Git identity."
    )


git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


print(
    "\nGit author:"
)

print(
    " ",
    author_name,
    "<" + author_email + ">"
)


# =============================================================================
# 20. COMMIT
# =============================================================================

banner(
    "STAGE26-6D :: COMMIT"
)


git(
    "add",
    str(
        CHECKPOINT_REL
    ),
)


print(
    git(
        "diff",
        "--cached",
        "--name-status",
    )
)


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-6D commit parent mismatch."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Stage26-6D commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository not clean after Stage26-6D commit."
    )


# =============================================================================
# 21. PUSH
# =============================================================================

banner(
    "STAGE26-6D :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    push_result = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
        text=True,
    )


    print(
        push_result.stdout.strip()
    )


github_token = None


# =============================================================================
# 22. REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-6D :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "origin/main did not advance to Stage26-6D."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote Stage26-6D parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote Stage26-6D subject mismatch."
    )


# =============================================================================
# 23. REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-6D :: REMOTE BYTE VERIFICATION"
)


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    str(
        MANIFEST_PATH.relative_to(
            REPO
        )
    ),
)


remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "Remote manifest SHA256:"
)

print(
    " ",
    remote_manifest_sha
)


if remote_manifest_sha != manifest_sha:

    raise RuntimeError(
        "Remote Stage26-6D manifest mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


if int(
    remote_manifest[
        "file_count_excluding_manifest"
    ]
) != 4:

    raise RuntimeError(
        "Unexpected Stage26-6D remote package size."
    )


for row in remote_manifest[
    "files"
]:

    data = git_blob_bytes(
        "origin/main",
        row[
            "repo_relative_path"
        ],
    )

    actual_size = len(
        data
    )

    actual_sha = sha256_bytes(
        data
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Remote Stage26-6D byte verification failed."
        )


# =============================================================================
# 24. REMOTE SCIENTIFIC AUDIT
# =============================================================================

banner(
    "STAGE26-6D :: REMOTE SCIENTIFIC AUDIT"
)


remote_frontier = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            FRONTIER_JSON.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_receipt = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            RECEIPT_PATH.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)


remote_checks = {
    "result PASS":
        (
            remote_frontier[
                "status"
            ]
            ==
            "PASS_WITHIN_GROUP_POINT_ESTIMATE_FRONTIERS"
        ),

    "Group A frontier":
        (
            remote_frontier[
                "groups"
            ][
                GROUP_A
            ][
                "frontier_members"
            ]
            ==
            EXPECTED_FRONTIER[
                GROUP_A
            ]
        ),

    "Group B frontier":
        (
            remote_frontier[
                "groups"
            ][
                GROUP_B
            ][
                "frontier_members"
            ]
            ==
            EXPECTED_FRONTIER[
                GROUP_B
            ]
        ),

    "Group B descriptive":
        (
            remote_frontier[
                "groups"
            ][
                GROUP_B
            ][
                "claim_boundary"
            ]
            ==
            "DESCRIPTIVE_NON_CONFIRMATORY"
        ),

    "14 ordered comparisons":
        (
            int(
                remote_frontier[
                    "pairwise_ordered_comparison_count"
                ]
            )
            ==
            14
        ),

    "zero cross-group":
        (
            int(
                remote_frontier[
                    "cross_group_comparison_count"
                ]
            )
            ==
            0
        ),

    "bootstrap false":
        (
            remote_receipt[
                "bootstrap_uncertainty_computed"
            ]
            is False
        ),

    "metric recomputation false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "predictive_metric_recomputed"
            ]
            is False
        ),

    "new inference false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "inference_performed"
            ]
            is False
        ),

    "new timing false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "timing_performed"
            ]
            is False
        ),

    "GPU false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "GPU_used"
            ]
            is False
        ),
}


for name, passed in remote_checks.items():

    print(
        f"{name:38s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    remote_checks.values()
):

    raise RuntimeError(
        "Remote Stage26-6D scientific audit failed."
    )


# =============================================================================
# 25. FINAL CLOSURE
# =============================================================================

banner(
    "STAGE26-6D CPU POINT-ESTIMATE PARETO COMPLETE"
)


final_status = git(
    "status",
    "--porcelain",
)


print(
    "NEW DURABLE COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nGROUP A FRONTIER:"
)

for target in frontier_by_group[
    GROUP_A
]:

    print(
        " ",
        target
    )


print(
    "\nGROUP A DOMINATED:"
)

for target in EXPECTED_GROUP_MEMBERS[
    GROUP_A
]:

    if target not in frontier_by_group[
        GROUP_A
    ]:

        print(
            " ",
            target,
            "<-",
            dominators[
                target
            ],
        )


print(
    "\nGROUP B FRONTIER — DESCRIPTIVE/NON-CONFIRMATORY:"
)

for target in frontier_by_group[
    GROUP_B
]:

    print(
        " ",
        target
    )


print(
    "\nGROUP B DOMINATED:"
)

for target in EXPECTED_GROUP_MEMBERS[
    GROUP_B
]:

    if target not in frontier_by_group[
        GROUP_B
    ]:

        print(
            " ",
            target,
            "<-",
            dominators[
                target
            ],
        )


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  point-estimate dominance computed : YES"
)

print(
    "  Group A frontier computed         : YES"
)

print(
    "  Group B frontier computed         : YES — descriptive only"
)

print(
    "  cross-group frontier              : NO"
)

print(
    "  bootstrap uncertainty             : NO"
)

print(
    "  predictive metric recomputation   : NO"
)

print(
    "  holdout reopened                  : NO"
)

print(
    "  model loading                     : NO"
)

print(
    "  new inference                     : NO"
)

print(
    "  new timing                        : NO"
)

print(
    "  GPU                               : NO"
)


print(
    "\nHASHES:"
)

print(
    "  pairwise CSV :",
    pairwise_sha
)

print(
    "  frontier JSON:",
    frontier_json_sha
)

print(
    "  frontier CSV :",
    frontier_csv_sha
)

print(
    "  receipt      :",
    receipt_sha
)

print(
    "  manifest     :",
    manifest_sha
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  commit             : PASS"
)

print(
    "  parent             : PASS"
)

print(
    "  package bytes      : PASS"
)

print(
    "  frontier content   : PASS"
)

print(
    "  scientific content : PASS"
)


print(
    "\nRepo clean:",
    final_status == ""
)


if final_status:

    raise RuntimeError(
        "Repository not clean after Stage26-6D."
    )


print(
    "\nNEXT:"
)

print(
    "  Stage26-6E: audit whether the frozen predictive and"
)

print(
    "  latency artifacts support the preregistered bootstrap"
)

print(
    "  uncertainty analysis without recomputation or leakage."
)

print(
    "  The Stage26-6D point-estimate frontiers are now immutable."
)

print(
    "  No GPU yet."
)


STAGE26-6D :: DURABLE SCIENTIFIC PARENT
Expected parent: 36a1767c81ccc44818c543e82c2b8da95be39560
Local HEAD     : 36a1767c81ccc44818c543e82c2b8da95be39560
origin/main    : 36a1767c81ccc44818c543e82c2b8da95be39560
Repo clean     : True

STAGE26-6D :: FROZEN INPUT IDENTITY
point lock             PASS 3b4739d9cec40c0dc1a21c80cb3cc4682cab08bef671f760de4697a7ad8d6ddf
rule lock              PASS aff5821c4e870bbd7326d93563c6e952cf34aafae06fa483e9ec6f348409a1d2
freeze receipt         PASS f86db80ce6a611963671603fb3a27b260649fd6af36610cb5d2fbcd920da90b8
lock manifest          PASS ddb1320491a54e4d6c5d461fb61c4854bfbf08e8f9585be2afff4ce8c8721479

STAGE26-6D :: FROZEN SCIENTIFIC INPUT GATE
Frozen point count : 6
Point-core SHA     : 5fb0e952d6560b6cd0f84c49048c05d26ff3c89c52edd61ea91855d374c10a76
Group A            : 4
Group B            : 2
Cross-group        : PROHIBITED
Bootstrap          : NOT PERFORMED

STAGE26-6D :: COMPUTE WITHIN-GROUP DOMINANCE

GROUP_A_DUPSAFE70
   STAGE16_XGBOOST_TUNE

In [17]:
# =============================================================================
# STAGE26-6E
# READ-ONLY BOOTSTRAP / UNCERTAINTY PROVENANCE AUDIT
#
# DURABLE SCIENTIFIC PARENT:
#   ff9d329785c6cd30d273729356f402e59dc4e844
#
# PURPOSE
# -------
# Audit exactly what uncertainty analysis is scientifically supported after
# the immutable Stage26-6D point-estimate Pareto result.
#
# QUESTIONS
# ---------
#
# 1. What exactly did the Stage26 protocol preregister for bootstrap?
#
# 2. Do the six primary CPU1/B1 Pareto latency conditions retain all raw
#    timing observations required for that preregistered bootstrap?
#
# 3. Do the existing Stage26-2 summaries already contain p95 bootstrap CIs?
#
# 4. Can we establish from committed artifacts/code HOW those existing
#    intervals were seeded?
#
# 5. Was PR-AUC itself preregistered as a Stage26 bootstrap target?
#
# 6. Do the exact frozen predictive source artifacts contain already-frozen
#    per-model PR-AUC uncertainty that could be reused without reopening data?
#
# IMPORTANT
# ---------
# THIS CELL DOES NOT YET:
#
#   - calculate a new bootstrap interval
#   - alter an existing bootstrap interval
#   - compute PR-AUC
#   - reopen predictive labels/predictions/holdout
#   - alter Stage26-6D frontier membership
#   - compute probabilistic Pareto membership
#   - load models
#   - run inference
#   - run timing
#   - access PCAP
#   - access Release corpus
#   - use GPU
#   - write anything into Git
#
# It creates ONE TRANSIENT audit outside the repository:
#
#   /kaggle/working/stage26_deployment_profiling/pareto/
#       stage26_6e_bootstrap_uncertainty_provenance_audit.json
#
# We inspect the output before deciding whether:
#
#   A. existing latency CIs are already protocol-compliant;
#   B. latency CIs require a derived-summary-only correction from preserved
#      raw timing observations;
#   C. predictive-axis bootstrap is unavailable / not preregistered and must
#      not be invented post hoc.
# =============================================================================

from __future__ import annotations

import csv
import io
import json
import os
import re
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. DURABLE IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "ff9d329785c6cd30d273729356f402e59dc4e844"
)


STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

PARETO_RUNTIME = (
    STAGE26_ROOT
    / "pareto"
)

OUT = (
    PARETO_RUNTIME
    / "stage26_6e_bootstrap_uncertainty_provenance_audit.json"
)


# -----------------------------------------------------------------------------
# Frozen global Stage26 protocol
# -----------------------------------------------------------------------------

PROTOCOL = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)


# -----------------------------------------------------------------------------
# Stage26-6C input lock
# -----------------------------------------------------------------------------

LOCK_6C = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_6c_pareto_point_lock"
)

POINT_LOCK = (
    LOCK_6C
    / "stage26_6c_resolved_pareto_points_lock.json"
)

RULE_LOCK = (
    LOCK_6C
    / "stage26_6c_pareto_rules_lock.json"
)

EXPECTED_POINT_LOCK_SHA256 = (
    "3b4739d9cec40c0dc1a21c80cb3cc4682cab08bef671f760de4697a7ad8d6ddf"
)

EXPECTED_RULE_LOCK_SHA256 = (
    "aff5821c4e870bbd7326d93563c6e952cf34aafae06fa483e9ec6f348409a1d2"
)

EXPECTED_POINT_SET_SHA256 = (
    "5fb0e952d6560b6cd0f84c49048c05d26ff3c89c52edd61ea91855d374c10a76"
)


# -----------------------------------------------------------------------------
# Stage26-6D immutable point-estimate result
# -----------------------------------------------------------------------------

DIR_6D = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_6d_cpu_pareto_point_estimate"
)

FRONTIER_6D = (
    DIR_6D
    / "stage26_6d_point_estimate_frontiers.json"
)

RECEIPT_6D = (
    DIR_6D
    / "stage26_6d_pareto_point_estimate_receipt.json"
)

MANIFEST_6D = (
    DIR_6D
    / "stage26_6d_pareto_point_estimate_manifest.json"
)

EXPECTED_FRONTIER_6D_SHA256 = (
    "367b3d34ddb4dd125dcbe3db71291b5576b4521640cd2f11d76e50dc247fc6b5"
)

EXPECTED_RECEIPT_6D_SHA256 = (
    "d898f08dc45cdb625efa5c000d76c4414a245d2091dc92274300f2260956bc79"
)

EXPECTED_MANIFEST_6D_SHA256 = (
    "7763a8bdb5c79fa2c94224866c95a43d7ad28a8f1756976b6a361d90bc7f1046"
)


# -----------------------------------------------------------------------------
# Stage26-2 preserved CPU timing observations / summaries
# -----------------------------------------------------------------------------

WARM_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_2_cpu_warm_inference"
)

WARM_RAW = (
    WARM_DIR
    / "stage26_2_warm_raw.csv"
)

WARM_SUMMARY = (
    WARM_DIR
    / "stage26_2_warm_summary.csv"
)

WARM_STATUS = (
    WARM_DIR
    / "stage26_2_condition_status.csv"
)

WARM_IMPLEMENTATION = (
    WARM_DIR
    / "stage26_2_warm_cpu_implementation.json"
)

WARM_RECEIPT = (
    WARM_DIR
    / "stage26_2_warm_cpu_receipt.json"
)


EXPECTED_WARM_SUMMARY_SHA256 = (
    "75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232"
)

EXPECTED_WARM_STATUS_SHA256 = (
    "df146563826992cb57702e589e4cf5d875ecfdbbf79533a731405e8eb738e7af"
)


# =============================================================================
# 1. EXPECTED PRIMARY CONDITIONS
# =============================================================================

PRIMARY_CONDITIONS = {
    "STAGE16_XGBOOST_TUNED":
        "CPUCOND_001",

    "STAGE16_LIGHTGBM_TUNED":
        "CPUCOND_011",

    "STAGE16_CATBOOST_TUNED":
        "CPUCOND_021",

    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING":
        "CPUCOND_031",

    "STAGE20_MASKED_CNN_V1":
        "CPUCOND_041",

    "STAGE21_MASKED_VIT_V1":
        "CPUCOND_051",
}


EXPECTED_GROUPS = {
    "STAGE16_XGBOOST_TUNED":
        "GROUP_A_DUPSAFE70",

    "STAGE16_LIGHTGBM_TUNED":
        "GROUP_A_DUPSAFE70",

    "STAGE16_CATBOOST_TUNED":
        "GROUP_A_DUPSAFE70",

    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING":
        "GROUP_A_DUPSAFE70",

    "STAGE20_MASKED_CNN_V1":
        "GROUP_B_PACKET_IMAGE",

    "STAGE21_MASKED_VIT_V1":
        "GROUP_B_PACKET_IMAGE",
}


# =============================================================================
# 2. FROZEN BOOTSTRAP EXPECTATION
# =============================================================================

EXPECTED_BOOTSTRAP = {
    "enabled":
        True,

    "interval":
        "PERCENTILE_95_PERCENT",

    "replicates":
        2000,

    "rng":
        "numpy.random.default_rng_PCG64",

    "seed":
        26042,
}


EXPECTED_BOOTSTRAP_TARGETS = [
    "p50_latency",
    "p95_latency",
    "p99_latency_when_n_gte_100",
    "median_throughput",
]


# =============================================================================
# 3. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
    text=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            + " ".join(
                map(
                    str,
                    cmd,
                )
            )
            + "\n\n"
            + output
        )

    return p


def git(
    *args,
):

    return run(
        [
            "git",
            *args,
        ],
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def load_csv(
    path,
):

    with Path(path).open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def unique(
    values,
):

    return sorted(
        set(
            values
        )
    )


def recursive_key_evidence(
    obj,
    *,
    path="$",
    predicate,
    out=None,
):

    if out is None:
        out = []


    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            child = (
                path
                + "."
                + str(
                    key
                )
            )


            if predicate(
                str(
                    key
                ),
                value,
            ):

                out.append(
                    {
                        "path":
                            child,

                        "value":
                            value,
                    }
                )


            recursive_key_evidence(
                value,
                path=child,
                predicate=predicate,
                out=out,
            )


    elif isinstance(
        obj,
        list,
    ):

        for index, value in enumerate(
            obj
        ):

            recursive_key_evidence(
                value,
                path=(
                    path
                    +
                    f"[{index}]"
                ),
                predicate=predicate,
                out=out,
            )


    return out


def strict_pr_auc_uncertainty_key(
    key,
    value,
):

    k = key.lower()


    has_pr = (
        "pr_auc" in k
        or
        "prauc" in k
        or
        "auprc" in k
        or
        "average_precision" in k
    )


    has_uncertainty = any(
        token in k
        for token in [
            "ci",
            "confidence",
            "bootstrap",
            "lower",
            "upper",
            "low",
            "high",
            "std",
            "stderr",
            "standard_error",
        ]
    )


    return (
        has_pr
        and
        has_uncertainty
    )


def generic_uncertainty_key(
    key,
    value,
):

    k = key.lower()

    return any(
        token in k
        for token in [
            "bootstrap",
            "confidence",
            "ci95",
            "ci_95",
            "lower",
            "upper",
            "stderr",
            "standard_error",
        ]
    )


# =============================================================================
# 4. DURABLE GIT STATE
# =============================================================================

banner(
    "STAGE26-6E :: DURABLE SCIENTIFIC STATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-6E parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before bootstrap audit."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if OUT.exists():

    raise RuntimeError(
        "Stage26-6E transient audit already exists. "
        "Stop for narrow recovery rather than overwrite."
    )


# =============================================================================
# 5. FROZEN 6C / 6D IDENTITY
# =============================================================================

banner(
    "STAGE26-6E :: IMMUTABLE PARETO INPUT / RESULT IDENTITY"
)


identity_checks = [
    (
        "Stage26 protocol",
        PROTOCOL,
        EXPECTED_PROTOCOL_SHA256,
    ),

    (
        "6C point lock",
        POINT_LOCK,
        EXPECTED_POINT_LOCK_SHA256,
    ),

    (
        "6C rule lock",
        RULE_LOCK,
        EXPECTED_RULE_LOCK_SHA256,
    ),

    (
        "6D frontier",
        FRONTIER_6D,
        EXPECTED_FRONTIER_6D_SHA256,
    ),

    (
        "6D receipt",
        RECEIPT_6D,
        EXPECTED_RECEIPT_6D_SHA256,
    ),

    (
        "6D manifest",
        MANIFEST_6D,
        EXPECTED_MANIFEST_6D_SHA256,
    ),

    (
        "Stage26-2 summary",
        WARM_SUMMARY,
        EXPECTED_WARM_SUMMARY_SHA256,
    ),

    (
        "Stage26-2 status",
        WARM_STATUS,
        EXPECTED_WARM_STATUS_SHA256,
    ),
]


identity_rows = []


for label, path, expected in identity_checks:

    if not path.is_file():

        raise FileNotFoundError(
            path
        )


    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:28s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    identity_rows.append(
        {
            "label":
                label,

            "path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "sha256":
                actual,

            "expected_sha256":
                expected,

            "passed":
                passed,
        }
    )


    if not passed:

        raise RuntimeError(
            f"Identity mismatch: {label}"
        )


warm_raw_sha = sha256_file(
    WARM_RAW
)

warm_impl_sha = sha256_file(
    WARM_IMPLEMENTATION
)

warm_receipt_sha = sha256_file(
    WARM_RECEIPT
)


print(
    "\nAdditional exact identities:"
)

print(
    "  warm raw CSV      :",
    warm_raw_sha
)

print(
    "  warm implementation:",
    warm_impl_sha
)

print(
    "  warm receipt      :",
    warm_receipt_sha
)


# =============================================================================
# 6. VERIFY STAGE26-6D REMAINS IMMUTABLE
# =============================================================================

banner(
    "STAGE26-6E :: POINT-ESTIMATE FRONTIER IMMUTABILITY"
)


frontier_6d = json.loads(
    FRONTIER_6D.read_text(
        encoding="utf-8"
    )
)

receipt_6d = json.loads(
    RECEIPT_6D.read_text(
        encoding="utf-8"
    )
)


if frontier_6d[
    "input_point_set_sha256"
] != EXPECTED_POINT_SET_SHA256:

    raise RuntimeError(
        "Stage26-6D point-set identity changed."
    )


if receipt_6d[
    "scientific_state"
][
    "point_estimate_frontiers_computed"
] is not True:

    raise RuntimeError(
        "Stage26-6D frontier is not complete."
    )


if receipt_6d[
    "scientific_state"
][
    "cross_group_frontier_computed"
] is not False:

    raise RuntimeError(
        "Unexpected cross-group frontier."
    )


if receipt_6d[
    "bootstrap_uncertainty_computed"
] is not False:

    raise RuntimeError(
        "Stage26-6D unexpectedly computed bootstrap uncertainty."
    )


print(
    "Point-estimate frontier committed : YES"
)

print(
    "Point-set SHA                    :",
    EXPECTED_POINT_SET_SHA256
)

print(
    "Cross-group frontier             : NO"
)

print(
    "Bootstrap in Stage26-6D          : NO"
)

print(
    "Stage26-6D membership may change : NO"
)


# =============================================================================
# 7. EXACT FROZEN STATISTICS PROTOCOL
# =============================================================================

banner(
    "STAGE26-6E :: FROZEN STATISTICS PROTOCOL"
)


protocol = json.loads(
    PROTOCOL.read_text(
        encoding="utf-8"
    )
)


statistics_protocol = protocol[
    "statistics_protocol"
]

bootstrap_protocol = statistics_protocol[
    "bootstrap"
]

bootstrap_targets = statistics_protocol[
    "bootstrap_targets"
]


print(
    "Bootstrap configuration:"
)

print(
    json.dumps(
        bootstrap_protocol,
        indent=2,
        sort_keys=True,
    )
)


print(
    "\nBootstrap targets:"
)

for target in bootstrap_targets:

    print(
        " ",
        target
    )


for key, expected in EXPECTED_BOOTSTRAP.items():

    actual = bootstrap_protocol[
        key
    ]


    if actual != expected:

        raise RuntimeError(
            f"Frozen bootstrap field changed: {key}\n"
            f"expected={expected!r}\n"
            f"actual={actual!r}"
        )


if bootstrap_targets != EXPECTED_BOOTSTRAP_TARGETS:

    raise RuntimeError(
        "Frozen bootstrap target list changed."
    )


predictive_metric_terms = {
    "pr_auc",
    "prauc",
    "auprc",
    "average_precision",
}


stage26_pr_auc_bootstrap_preregistered = any(
    any(
        term in str(
            target
        ).lower()
        for term in predictive_metric_terms
    )
    for target in bootstrap_targets
)


print(
    "\nFrozen seed       :",
    bootstrap_protocol[
        "seed"
    ]
)

print(
    "Frozen replicates :",
    bootstrap_protocol[
        "replicates"
    ]
)

print(
    "Frozen interval   :",
    bootstrap_protocol[
        "interval"
    ]
)

print(
    "Frozen RNG        :",
    bootstrap_protocol[
        "rng"
    ]
)

print(
    "PR-AUC listed as bootstrap target:",
    stage26_pr_auc_bootstrap_preregistered
)


if stage26_pr_auc_bootstrap_preregistered:

    raise RuntimeError(
        "Unexpected: PR-AUC is now present in frozen Stage26 bootstrap targets."
    )


# =============================================================================
# 8. PARETO RULE WORDING
# =============================================================================

banner(
    "STAGE26-6E :: PARETO UNCERTAINTY WORDING"
)


pareto = protocol[
    "pareto_protocol"
]


frontier_status = pareto[
    "dominance_rule"
][
    "frontier_status"
]


print(
    "Frozen frontier status:"
)

print(
    " ",
    frontier_status
)


if frontier_status != (
    "Descriptive point-estimate frontier; bootstrap uncertainty "
    "reported separately."
):

    raise RuntimeError(
        "Frozen Pareto frontier-status wording changed."
    )


print(
    "\nInterpretation audit:"
)

print(
    "  point-estimate frontier is separate from uncertainty : YES"
)

print(
    "  bootstrap targets explicitly include latency         : YES"
)

print(
    "  bootstrap targets explicitly include PR-AUC          : NO"
)

print(
    "  therefore PR-AUC bootstrap may NOT be silently added : YES"
)


# =============================================================================
# 9. LOAD EXACT FROZEN SIX POINTS
# =============================================================================

banner(
    "STAGE26-6E :: SIX PRIMARY COST CONDITIONS"
)


point_lock = json.loads(
    POINT_LOCK.read_text(
        encoding="utf-8"
    )
)


if point_lock[
    "point_set_sha256"
] != EXPECTED_POINT_SET_SHA256:

    raise RuntimeError(
        "6C point-set SHA mismatch."
    )


points = point_lock[
    "points"
]


if len(
    points
) != 6:

    raise RuntimeError(
        "Expected six frozen primary points."
    )


for point in points:

    target = point[
        "target_id"
    ]

    condition_id = point[
        "cost_source"
    ][
        "condition_id"
    ]


    if PRIMARY_CONDITIONS.get(
        target
    ) != condition_id:

        raise RuntimeError(
            f"Unexpected CPU1/B1 condition mapping for {target}."
        )


    if point[
        "comparison_group"
    ] != EXPECTED_GROUPS[
        target
    ]:

        raise RuntimeError(
            f"Unexpected group for {target}."
        )


    print(
        f"{target:42s} "
        f"{condition_id:11s} "
        f"p95={float(point['p95_cpu1_batch1_inference_latency_ms']):.12f} ms"
    )


# =============================================================================
# 10. STAGE26-2 SUMMARY CI STRUCTURE
# =============================================================================

banner(
    "STAGE26-6E :: EXISTING STAGE26-2 P95 CI INVENTORY"
)


summary_rows = load_csv(
    WARM_SUMMARY
)


if not summary_rows:

    raise RuntimeError(
        "Stage26-2 summary is empty."
    )


summary_columns = list(
    summary_rows[
        0
    ].keys()
)


print(
    "Summary columns:"
)

for column in summary_columns:

    print(
        " ",
        column
    )


required_ci_columns = [
    "p95_batch_latency_ms",
    "p95_ci95_low_ms",
    "p95_ci95_high_ms",
]


for column in required_ci_columns:

    if column not in summary_columns:

        raise RuntimeError(
            f"Required Stage26-2 p95/CI column absent: {column}"
        )


summary_by_condition = {}


for target, condition_id in PRIMARY_CONDITIONS.items():

    matches = [
        row
        for row in summary_rows
        if (
            row[
                "condition_id"
            ]
            ==
            condition_id
            and
            row[
                "target_id"
            ]
            ==
            target
            and
            row[
                "hardware_mode"
            ]
            ==
            "CPU_1_PHYSICAL_CORE"
            and
            int(
                row[
                    "batch_size"
                ]
            )
            ==
            1
            and
            int(
                row[
                    "thread_count"
                ]
            )
            ==
            1
        )
    ]


    if len(
        matches
    ) != 1:

        raise RuntimeError(
            f"{target}: expected one CPU1/B1 Stage26-2 summary row; "
            f"found {len(matches)}."
        )


    row = matches[
        0
    ]


    n = int(
        row[
            "n"
        ]
    )

    p95 = float(
        row[
            "p95_batch_latency_ms"
        ]
    )

    ci_low = float(
        row[
            "p95_ci95_low_ms"
        ]
    )

    ci_high = float(
        row[
            "p95_ci95_high_ms"
        ]
    )


    if n != 200:

        raise RuntimeError(
            f"{target}: expected n=200 for CPU1/B1."
        )


    if ci_low > p95:

        raise RuntimeError(
            f"{target}: stored p95 CI lower bound exceeds point estimate."
        )


    if ci_high < p95:

        raise RuntimeError(
            f"{target}: stored p95 CI upper bound is below point estimate."
        )


    summary_by_condition[
        condition_id
    ] = {
        "target_id":
            target,

        "n":
            n,

        "p95_batch_latency_ms":
            p95,

        "p95_ci95_low_ms":
            ci_low,

        "p95_ci95_high_ms":
            ci_high,
    }


    print(
        f"{target:42s} "
        f"p95={p95:12.9f} ms "
        f"CI95=[{ci_low:12.9f}, {ci_high:12.9f}]"
    )


print(
    "\nExisting p95 CI columns present for all six:",
    len(
        summary_by_condition
    )
    ==
    6
)


# =============================================================================
# 11. RAW TIMING SUPPORT FOR PREREGISTERED BOOTSTRAP
# =============================================================================

banner(
    "STAGE26-6E :: RAW TIMING OBSERVATION SUPPORT"
)


raw_rows = load_csv(
    WARM_RAW
)


if not raw_rows:

    raise RuntimeError(
        "Stage26-2 raw timing CSV is empty."
    )


raw_columns = list(
    raw_rows[
        0
    ].keys()
)


expected_raw_columns = [
    "execution_order",
    "condition_id",
    "target_id",
    "target_role",
    "comparison_group",
    "hardware_mode",
    "thread_count",
    "batch_size",
    "iteration_index",
    "elapsed_ns",
    "amortized_ns_per_flow",
    "flows_per_second",
    "status",
]


missing_raw_columns = [
    column
    for column in expected_raw_columns
    if column not in raw_columns
]


if missing_raw_columns:

    raise RuntimeError(
        "Stage26-2 raw timing schema missing columns:\n"
        +
        "\n".join(
            missing_raw_columns
        )
    )


raw_support = {}


for target, condition_id in PRIMARY_CONDITIONS.items():

    matches = [
        row
        for row in raw_rows
        if (
            row[
                "condition_id"
            ]
            ==
            condition_id
            and
            row[
                "target_id"
            ]
            ==
            target
        )
    ]


    if len(
        matches
    ) != 200:

        raise RuntimeError(
            f"{target}: expected exactly 200 raw timing rows; "
            f"found {len(matches)}."
        )


    if unique(
        row[
            "hardware_mode"
        ]
        for row in matches
    ) != [
        "CPU_1_PHYSICAL_CORE"
    ]:

        raise RuntimeError(
            f"{target}: raw observations are not all CPU1."
        )


    if unique(
        int(
            row[
                "thread_count"
            ]
        )
        for row in matches
    ) != [
        1
    ]:

        raise RuntimeError(
            f"{target}: raw observations do not all use one thread."
        )


    if unique(
        int(
            row[
                "batch_size"
            ]
        )
        for row in matches
    ) != [
        1
    ]:

        raise RuntimeError(
            f"{target}: raw observations are not all B=1."
        )


    if unique(
        row[
            "status"
        ]
        for row in matches
    ) != [
        "PASS"
    ]:

        raise RuntimeError(
            f"{target}: raw observations contain non-PASS rows."
        )


    iteration_indices = sorted(
        int(
            row[
                "iteration_index"
            ]
        )
        for row in matches
    )


    if iteration_indices != list(
        range(
            1,
            201,
        )
    ):

        raise RuntimeError(
            f"{target}: raw iteration indices are not exactly 1..200."
        )


    elapsed_ns = [
        int(
            row[
                "elapsed_ns"
            ]
        )
        for row in matches
    ]


    if any(
        value <= 0
        for value in elapsed_ns
    ):

        raise RuntimeError(
            f"{target}: non-positive raw timing observation."
        )


    raw_support[
        target
    ] = {
        "condition_id":
            condition_id,

        "observation_count":
            len(
                elapsed_ns
            ),

        "iteration_min":
            min(
                iteration_indices
            ),

        "iteration_max":
            max(
                iteration_indices
            ),

        "elapsed_ns_min":
            min(
                elapsed_ns
            ),

        "elapsed_ns_max":
            max(
                elapsed_ns
            ),

        "raw_observations_sufficient_for_latency_bootstrap":
            True,
    }


    print(
        f"{target:42s} "
        f"{condition_id:11s} "
        f"n={len(elapsed_ns):3d} "
        f"elapsed_ns=[{min(elapsed_ns)}, {max(elapsed_ns)}] "
        f"PASS"
    )


latency_raw_support_complete = all(
    record[
        "raw_observations_sufficient_for_latency_bootstrap"
    ]
    for record in raw_support.values()
)


print(
    "\nAll six preserve raw B1 timing observations:",
    latency_raw_support_complete
)


# =============================================================================
# 12. CHECK POINT ESTIMATE AGAINST RAW TIMING — NOT BOOTSTRAP
# =============================================================================

banner(
    "STAGE26-6E :: RAW / SUMMARY POINT-ESTIMATE CONSISTENCY"
)


# Computing a deterministic percentile from the already-measured observations
# is ONLY an integrity check of the existing p95 point estimate.
# It is NOT a bootstrap and creates no uncertainty result.

def linear_percentile_95_ns(
    values,
):

    values = sorted(
        int(
            value
        )
        for value in values
    )


    n = len(
        values
    )

    if n == 0:

        raise RuntimeError(
            "Cannot compute percentile of empty values."
        )


    # NumPy percentile(method="linear") semantics:
    # h = (n - 1) * q
    q = 0.95

    h = (
        n - 1
    ) * q

    lo = int(
        h
        // 1
    )

    hi = min(
        lo + 1,
        n - 1,
    )

    fraction = (
        h
        -
        lo
    )


    return (
        values[
            lo
        ]
        +
        fraction
        *
        (
            values[
                hi
            ]
            -
            values[
                lo
            ]
        )
    )


point_estimate_checks = {}


for target, condition_id in PRIMARY_CONDITIONS.items():

    matches = [
        row
        for row in raw_rows
        if (
            row[
                "condition_id"
            ]
            ==
            condition_id
            and
            row[
                "target_id"
            ]
            ==
            target
        )
    ]


    elapsed_ns = [
        int(
            row[
                "elapsed_ns"
            ]
        )
        for row in matches
    ]


    raw_p95_ms = (
        linear_percentile_95_ns(
            elapsed_ns
        )
        /
        1_000_000.0
    )


    stored_p95_ms = summary_by_condition[
        condition_id
    ][
        "p95_batch_latency_ms"
    ]


    exact = (
        raw_p95_ms
        ==
        stored_p95_ms
    )


    point_estimate_checks[
        target
    ] = {
        "raw_rederived_p95_ms":
            raw_p95_ms,

        "stored_p95_ms":
            stored_p95_ms,

        "exact_match":
            exact,
    }


    print(
        f"{target:42s} "
        f"raw={repr(raw_p95_ms):22s} "
        f"stored={repr(stored_p95_ms):22s} "
        f"{'PASS' if exact else 'FAIL'}"
    )


    if not exact:

        raise RuntimeError(
            f"{target}: raw p95 does not exactly reproduce Stage26-2 point estimate."
        )


# =============================================================================
# 13. AUDIT COMMITTED BOOTSTRAP-SEED / IMPLEMENTATION EVIDENCE
# =============================================================================

banner(
    "STAGE26-6E :: COMMITTED BOOTSTRAP IMPLEMENTATION EVIDENCE"
)


# We are NOT executing any discovered implementation.
# We are only reading tracked text/code to see whether the bootstrap seed
# semantics are explicitly documented/committed.

grep_patterns = [
    "stable_seed",
    "bootstrap_seed",
    "bootstrap_ci",
    "bootstrap_interval",
    "default_rng",
    "replicates",
]


grep_evidence = []


for pattern in grep_patterns:

    p = run(
        [
            "git",
            "grep",
            "-n",
            "-I",
            "-e",
            pattern,
            "--",
            "*.py",
            "*.json",
            "*.md",
            "*.txt",
        ],
        check=False,
        text=True,
    )


    if p.returncode not in {
        0,
        1,
    }:

        raise RuntimeError(
            f"git grep failed for pattern {pattern!r}:\n"
            +
            p.stdout
        )


    for line in p.stdout.splitlines():

        # Keep Stage26-relevant evidence preferentially.
        lower = line.lower()

        if (
            "stage26"
            not in lower
            and
            "bootstrap"
            not in lower
            and
            "stable_seed"
            not in lower
        ):

            continue


        grep_evidence.append(
            {
                "pattern":
                    pattern,

                "evidence":
                    line[
                        :4000
                    ],
            }
        )


# Deduplicate while preserving order.
seen = set()

deduped_grep_evidence = []


for item in grep_evidence:

    key = item[
        "evidence"
    ]


    if key in seen:
        continue


    seen.add(
        key
    )

    deduped_grep_evidence.append(
        item
    )


grep_evidence = deduped_grep_evidence


print(
    "Relevant committed grep evidence rows:",
    len(
        grep_evidence
    )
)


for item in grep_evidence[
    :80
]:

    print(
        "\n",
        item[
            "evidence"
        ],
        sep="",
    )


if len(
    grep_evidence
) > 80:

    print(
        f"\n... {len(grep_evidence) - 80} additional evidence rows "
        "stored in transient audit."
    )


stable_seed_evidence = [
    item
    for item in grep_evidence
    if "stable_seed" in item[
        "evidence"
    ]
]


explicit_literal_seed_evidence = [
    item
    for item in grep_evidence
    if (
        "26042"
        in
        item[
            "evidence"
        ]
        and
        (
            "bootstrap"
            in
            item[
                "evidence"
            ].lower()
            or
            "default_rng"
            in
            item[
                "evidence"
            ].lower()
        )
    )
]


print(
    "\nEvidence containing stable_seed:",
    len(
        stable_seed_evidence
    )
)

print(
    "Evidence explicitly coupling bootstrap/RNG with literal 26042:",
    len(
        explicit_literal_seed_evidence
    )
)


# =============================================================================
# 14. STRUCTURED STAGE26-2 IMPLEMENTATION/RECEIPT EVIDENCE
# =============================================================================

banner(
    "STAGE26-6E :: STAGE26-2 STRUCTURED SEED EVIDENCE"
)


warm_implementation = json.loads(
    WARM_IMPLEMENTATION.read_text(
        encoding="utf-8"
    )
)

warm_receipt = json.loads(
    WARM_RECEIPT.read_text(
        encoding="utf-8"
    )
)


structured_impl_evidence = []


for label, obj in [
    (
        "stage26_2_warm_cpu_implementation",
        warm_implementation,
    ),

    (
        "stage26_2_warm_cpu_receipt",
        warm_receipt,
    ),
]:

    evidence = recursive_key_evidence(
        obj,
        predicate=lambda key, value: (
            "bootstrap"
            in
            key.lower()
            or
            "seed"
            in
            key.lower()
        ),
    )


    structured_impl_evidence.append(
        {
            "artifact":
                label,

            "evidence":
                evidence,
        }
    )


    print(
        "\n",
        label,
        sep="",
    )


    if not evidence:

        print(
            "  no bootstrap/seed metadata fields"
        )


    else:

        for item in evidence:

            print(
                " ",
                item[
                    "path"
                ],
                "=",
                repr(
                    item[
                        "value"
                    ]
                ),
            )


# =============================================================================
# 15. PREDICTIVE-AXIS UNCERTAINTY AVAILABILITY
# =============================================================================

banner(
    "STAGE26-6E :: FROZEN PREDICTIVE-SOURCE UNCERTAINTY AUDIT"
)


predictive_source_audit = {}


for point in points:

    target = point[
        "target_id"
    ]

    predictive = point[
        "predictive_source"
    ]

    source_rel = predictive[
        "source_path"
    ]

    source_path = (
        REPO
        /
        source_rel
    )


    if not source_path.is_file():

        raise FileNotFoundError(
            source_path
        )


    expected_source_sha = predictive[
        "source_sha256"
    ]

    actual_source_sha = sha256_file(
        source_path
    )


    if actual_source_sha != expected_source_sha:

        raise RuntimeError(
            f"Predictive source hash changed for {target}."
        )


    suffix = source_path.suffix.lower()

    strict_pr_uncertainty = []

    generic_uncertainty = []


    if suffix == ".json":

        obj = json.loads(
            source_path.read_text(
                encoding="utf-8"
            )
        )


        strict_pr_uncertainty = recursive_key_evidence(
            obj,
            predicate=strict_pr_auc_uncertainty_key,
        )


        generic_uncertainty = recursive_key_evidence(
            obj,
            predicate=generic_uncertainty_key,
        )


    elif suffix == ".csv":

        with source_path.open(
            "r",
            encoding="utf-8",
            newline="",
        ) as f:

            reader = csv.reader(
                f
            )

            header = next(
                reader
            )


        strict_pr_uncertainty = [
            {
                "column":
                    column,
            }
            for column in header
            if strict_pr_auc_uncertainty_key(
                column,
                None,
            )
        ]


        generic_uncertainty = [
            {
                "column":
                    column,
            }
            for column in header
            if generic_uncertainty_key(
                column,
                None,
            )
        ]


    else:

        text = source_path.read_text(
            encoding="utf-8",
            errors="replace",
        )


        for line_number, line in enumerate(
            text.splitlines(),
            start=1,
        ):

            lower = line.lower()


            if (
                (
                    "pr_auc"
                    in
                    lower
                    or
                    "auprc"
                    in
                    lower
                    or
                    "average_precision"
                    in
                    lower
                )
                and
                any(
                    token in lower
                    for token in [
                        "bootstrap",
                        "ci",
                        "confidence",
                        "lower",
                        "upper",
                    ]
                )
            ):

                strict_pr_uncertainty.append(
                    {
                        "line":
                            line_number,

                        "text":
                            line[
                                :2000
                            ],
                    }
                )


    predictive_source_audit[
        target
    ] = {
        "source_path":
            source_rel,

        "source_sha256":
            actual_source_sha,

        "source_field":
            predictive.get(
                "source_field"
            ),

        "pr_auc_value":
            float(
                point[
                    "pr_auc"
                ]
            ),

        "strict_per_model_pr_auc_uncertainty_evidence":
            strict_pr_uncertainty,

        "generic_uncertainty_evidence":
            generic_uncertainty,

        "strict_per_model_pr_auc_uncertainty_found":
            bool(
                strict_pr_uncertainty
            ),
    }


    print(
        f"\n{target}"
    )

    print(
        "  source:",
        source_rel
    )

    print(
        "  frozen PR-AUC:",
        repr(
            float(
                point[
                    "pr_auc"
                ]
            )
        )
    )

    print(
        "  strict per-model PR-AUC uncertainty evidence:",
        len(
            strict_pr_uncertainty
        )
    )


    for item in strict_pr_uncertainty[
        :10
    ]:

        print(
            "   ",
            item
        )


    if generic_uncertainty:

        print(
            "  generic uncertainty fields in source:",
            len(
                generic_uncertainty
            ),
            "(not automatically treated as per-model PR-AUC uncertainty)"
        )


all_six_have_existing_strict_pr_auc_uncertainty = all(
    record[
        "strict_per_model_pr_auc_uncertainty_found"
    ]
    for record in predictive_source_audit.values()
)


print(
    "\nAll six have frozen per-model PR-AUC uncertainty:",
    all_six_have_existing_strict_pr_auc_uncertainty
)


# =============================================================================
# 16. SCIENTIFIC CLASSIFICATION
# =============================================================================

banner(
    "STAGE26-6E :: SCIENTIFIC CLASSIFICATION"
)


# Timing-axis support is straightforward:
# the protocol preregistered timing bootstrap and raw timing observations exist.
timing_axis_bootstrap_supported = (
    bootstrap_protocol[
        "enabled"
    ]
    is True
    and
    "p95_latency"
    in
    bootstrap_targets
    and
    latency_raw_support_complete
)


# Predictive-axis bootstrap is NOT allowed merely because the word
# "bootstrap" appears in Pareto wording. It must have been preregistered or
# already exist as exact frozen source uncertainty.
predictive_axis_new_bootstrap_allowed = (
    stage26_pr_auc_bootstrap_preregistered
)


predictive_axis_existing_uncertainty_complete = (
    all_six_have_existing_strict_pr_auc_uncertainty
)


# Can we certify the current Stage26-2 CI seed provenance from committed
# implementation evidence?
#
# This cell does not decide compliance merely from a lack of evidence.
# It classifies provenance conservatively.

if stable_seed_evidence:

    stored_latency_ci_seed_status = (
        "STABLE_SEED_IMPLEMENTATION_EVIDENCE_FOUND__"
        "REQUIRES_EXACT_REVIEW_BEFORE_ACCEPTANCE"
    )

elif explicit_literal_seed_evidence:

    stored_latency_ci_seed_status = (
        "LITERAL_26042_BOOTSTRAP_RNG_EVIDENCE_FOUND__"
        "REQUIRES_EXACT_REVIEW_BEFORE_ACCEPTANCE"
    )

else:

    stored_latency_ci_seed_status = (
        "SEED_PROVENANCE_NOT_EXPLICITLY_CERTIFIED_FROM_COMMITTED_EVIDENCE"
    )


if not timing_axis_bootstrap_supported:

    overall_uncertainty_status = (
        "BLOCKED_TIMING_BOOTSTRAP_INPUTS_INCOMPLETE"
    )

elif (
    stable_seed_evidence
    or
    not explicit_literal_seed_evidence
):

    overall_uncertainty_status = (
        "TIMING_BOOTSTRAP_RAW_DATA_AVAILABLE__"
        "EXISTING_CI_SEED_PROVENANCE_REQUIRES_RESOLUTION"
    )

else:

    overall_uncertainty_status = (
        "TIMING_BOOTSTRAP_RAW_DATA_AVAILABLE__"
        "EXISTING_CI_PROVENANCE_CANDIDATE_FOR_ACCEPTANCE"
    )


print(
    "Timing p95 bootstrap preregistered:",
    (
        "p95_latency"
        in
        bootstrap_targets
    )
)

print(
    "Timing raw observations complete:",
    latency_raw_support_complete
)

print(
    "Existing p95 CI columns present:",
    len(
        summary_by_condition
    )
    ==
    6
)

print(
    "Stored CI seed provenance status:"
)

print(
    " ",
    stored_latency_ci_seed_status
)


print(
    "\nPR-AUC bootstrap preregistered by Stage26:",
    stage26_pr_auc_bootstrap_preregistered
)

print(
    "All six have pre-existing frozen per-model PR-AUC uncertainty:",
    predictive_axis_existing_uncertainty_complete
)

print(
    "New Stage26 PR-AUC bootstrap permitted from current protocol:",
    predictive_axis_new_bootstrap_allowed
)


print(
    "\nOVERALL UNCERTAINTY AUDIT STATUS:"
)

print(
    " ",
    overall_uncertainty_status
)


# =============================================================================
# 17. WRITE TRANSIENT AUDIT
# =============================================================================

banner(
    "STAGE26-6E :: WRITE TRANSIENT AUDIT"
)


PARETO_RUNTIME.mkdir(
    parents=True,
    exist_ok=True,
)


audit = {
    "schema":
        "stage26_6e_bootstrap_uncertainty_provenance_audit_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "status":
        overall_uncertainty_status,

    "immutable_point_estimate_result": {
        "stage26_6d_frontier_sha256":
            EXPECTED_FRONTIER_6D_SHA256,

        "stage26_6d_receipt_sha256":
            EXPECTED_RECEIPT_6D_SHA256,

        "stage26_6d_manifest_sha256":
            EXPECTED_MANIFEST_6D_SHA256,

        "point_set_sha256":
            EXPECTED_POINT_SET_SHA256,

        "frontier_membership_may_be_modified_by_this_audit":
            False,
    },

    "protocol": {
        "measurement_protocol_sha256":
            EXPECTED_PROTOCOL_SHA256,

        "statistics_protocol":
            statistics_protocol,

        "bootstrap":
            bootstrap_protocol,

        "bootstrap_targets":
            bootstrap_targets,

        "pr_auc_bootstrap_preregistered":
            stage26_pr_auc_bootstrap_preregistered,
    },

    "stage26_2_inputs": {
        "warm_raw_path":
            str(
                WARM_RAW.relative_to(
                    REPO
                )
            ),

        "warm_raw_sha256":
            warm_raw_sha,

        "warm_summary_path":
            str(
                WARM_SUMMARY.relative_to(
                    REPO
                )
            ),

        "warm_summary_sha256":
            EXPECTED_WARM_SUMMARY_SHA256,

        "warm_status_sha256":
            EXPECTED_WARM_STATUS_SHA256,

        "warm_implementation_sha256":
            warm_impl_sha,

        "warm_receipt_sha256":
            warm_receipt_sha,
    },

    "six_primary_latency_conditions": {
        "summary":
            summary_by_condition,

        "raw_support":
            raw_support,

        "point_estimate_integrity_checks":
            point_estimate_checks,

        "all_six_raw_observation_sets_complete":
            latency_raw_support_complete,

        "all_six_existing_p95_CIs_present":
            (
                len(
                    summary_by_condition
                )
                ==
                6
            ),
    },

    "committed_bootstrap_implementation_evidence": {
        "grep_evidence":
            grep_evidence,

        "stable_seed_evidence_count":
            len(
                stable_seed_evidence
            ),

        "literal_seed_26042_bootstrap_rng_evidence_count":
            len(
                explicit_literal_seed_evidence
            ),

        "structured_stage26_2_evidence":
            structured_impl_evidence,

        "stored_latency_ci_seed_status":
            stored_latency_ci_seed_status,
    },

    "predictive_axis_uncertainty": {
        "new_stage26_pr_auc_bootstrap_preregistered":
            stage26_pr_auc_bootstrap_preregistered,

        "new_stage26_pr_auc_bootstrap_allowed":
            predictive_axis_new_bootstrap_allowed,

        "frozen_predictive_source_audit":
            predictive_source_audit,

        "all_six_have_existing_strict_per_model_pr_auc_uncertainty":
            predictive_axis_existing_uncertainty_complete,
    },

    "scientific_conclusions": {
        "point_estimate_frontier_remains_immutable":
            True,

        "timing_axis_p95_bootstrap_is_preregistered":
            (
                "p95_latency"
                in
                bootstrap_targets
            ),

        "timing_axis_raw_data_support_bootstrap":
            latency_raw_support_complete,

        "existing_latency_CIs_may_be_used_without_seed_provenance_audit":
            False,

        "PR_AUC_is_not_a_stage26_bootstrap_target":
            (
                not
                stage26_pr_auc_bootstrap_preregistered
            ),

        "post_hoc_PR_AUC_bootstrap_may_be_invented":
            False,

        "cross_group_uncertainty_comparison_allowed":
            False,
    },

    "scientific_state": {
        "new_bootstrap_interval_computed":
            False,

        "existing_bootstrap_interval_modified":
            False,

        "PR_AUC_recomputed":
            False,

        "holdout_reopened":
            False,

        "predictive_scores_loaded":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "pareto_frontier_recomputed":
            False,

        "cross_group_frontier_computed":
            False,

        "PCAP_accessed":
            False,

        "Release_corpus_accessed":
            False,

        "GPU_used":
            False,

        "Git_modified":
            False,
    },

    "next": (
        "Review seed-provenance evidence. If existing Stage26-2 bootstrap CIs "
        "do not demonstrably implement the frozen seed=26042 semantics, freeze "
        "a derived-summary-only correction protocol BEFORE recomputing any "
        "latency CI from the immutable raw timing observations. Do not create "
        "new PR-AUC bootstrap uncertainty unless it was already prospectively "
        "specified by a frozen source."
    ),
}


with OUT.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        audit,
        f,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    )

    f.write(
        "\n"
    )


out_sha = sha256_file(
    OUT
)


print(
    "Audit:"
)

print(
    " ",
    OUT
)

print(
    "SHA256:"
)

print(
    " ",
    out_sha
)


# =============================================================================
# 18. FINAL READ-ONLY AUDIT
# =============================================================================

banner(
    "STAGE26-6E BOOTSTRAP / UNCERTAINTY PROVENANCE AUDIT COMPLETE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:

    raise RuntimeError(
        "Git HEAD changed during Stage26-6E."
    )


if final_remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed during Stage26-6E."
    )


if final_status:

    raise RuntimeError(
        "Stage26-6E unexpectedly modified Git."
    )


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  Stage26-6D point-estimate frontier immutable : YES"
)

print(
    "  raw CPU1/B1 timing observations audited       : YES"
)

print(
    "  existing p95 CI fields inventoried            : YES"
)

print(
    "  bootstrap seed provenance inspected           : YES"
)

print(
    "  new bootstrap interval computed               : NO"
)

print(
    "  existing bootstrap interval modified          : NO"
)

print(
    "  PR-AUC bootstrap preregistered                 :",
    stage26_pr_auc_bootstrap_preregistered
)

print(
    "  PR-AUC recomputed                              : NO"
)

print(
    "  holdout reopened                               : NO"
)

print(
    "  predictive scores loaded                       : NO"
)

print(
    "  model loaded                                   : NO"
)

print(
    "  inference performed                            : NO"
)

print(
    "  timing performed                               : NO"
)

print(
    "  cross-group frontier                           : NO"
)

print(
    "  GPU                                            : NO"
)

print(
    "  Git modified                                   : NO"
)


print(
    "\nAUDIT STATUS:"
)

print(
    " ",
    overall_uncertainty_status
)


print(
    "\nNEXT AFTER REVIEWING OUTPUT:"
)

print(
    "  If committed evidence shows derived/non-literal bootstrap seeds,"
)

print(
    "  freeze a pre-correction statistical implementation lock and"
)

print(
    "  regenerate ONLY derived timing uncertainty summaries from the"
)

print(
    "  already-preserved raw Stage26 observations using the frozen"
)

print(
    "  2,000-replicate PCG64 seed=26042 protocol."
)

print(
    "  No model remeasurement is permitted or necessary."
)

print(
    "  Do not invent a new PR-AUC bootstrap post hoc."
)

print(
    "  No GPU yet."
)


STAGE26-6E :: DURABLE SCIENTIFIC STATE
Expected parent: ff9d329785c6cd30d273729356f402e59dc4e844
Local HEAD     : ff9d329785c6cd30d273729356f402e59dc4e844
origin/main    : ff9d329785c6cd30d273729356f402e59dc4e844
Repo clean     : True

STAGE26-6E :: IMMUTABLE PARETO INPUT / RESULT IDENTITY
Stage26 protocol             PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
6C point lock                PASS 3b4739d9cec40c0dc1a21c80cb3cc4682cab08bef671f760de4697a7ad8d6ddf
6C rule lock                 PASS aff5821c4e870bbd7326d93563c6e952cf34aafae06fa483e9ec6f348409a1d2
6D frontier                  PASS 367b3d34ddb4dd125dcbe3db71291b5576b4521640cd2f11d76e50dc247fc6b5
6D receipt                   PASS d898f08dc45cdb625efa5c000d76c4414a245d2091dc92274300f2260956bc79
6D manifest                  PASS 7763a8bdb5c79fa2c94224866c95a43d7ad28a8f1756976b6a361d90bc7f1046
Stage26-2 summary            PASS 75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232
Stage26-2 statu

RuntimeError: STAGE16_XGBOOST_TUNED: raw p95 does not exactly reproduce Stage26-2 point estimate.

In [18]:
# =============================================================================
# STAGE26-6E-DIAG
# NARROW DIAGNOSIS OF RAW -> STORED P95 BINARY64 SEMANTICS
#
# PURPOSE
# -------
# Determine the exact computation path that produced the already-frozen
# Stage26-2 p95 point estimates.
#
# We test, WITHOUT ALTERING ANY RESULT:
#
#   A. np.percentile(elapsed_ns, 95) / 1e6
#   B. np.percentile(elapsed_ns / 1e6, 95)
#   C. np.quantile(elapsed_ns, 0.95) / 1e6
#   D. np.quantile(elapsed_ns / 1e6, 0.95)
#
# using NumPy method="linear".
#
# We also inspect committed Stage26 code/text for the exact percentile
# implementation used historically.
#
# NO WRITES
# NO BOOTSTRAP
# NO NEW INTERVALS
# NO PR-AUC
# NO HOLDOUT
# NO MODEL
# NO INFERENCE
# NO TIMING
# NO PARETO CHANGE
# NO GPU
# =============================================================================

from __future__ import annotations

import csv
import json
import hashlib
import subprocess
from pathlib import Path

import numpy as np


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "ff9d329785c6cd30d273729356f402e59dc4e844"
)


WARM_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_2_cpu_warm_inference"
)

WARM_RAW = (
    WARM_DIR
    / "stage26_2_warm_raw.csv"
)

WARM_SUMMARY = (
    WARM_DIR
    / "stage26_2_warm_summary.csv"
)


EXPECTED_WARM_RAW_SHA256 = (
    "78c58289ccfc4598966d6516201028f43c4b1d16d8bd0268a0cf58129d4179fa"
)

EXPECTED_WARM_SUMMARY_SHA256 = (
    "75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232"
)


PRIMARY_CONDITIONS = {
    "STAGE16_XGBOOST_TUNED":
        "CPUCOND_001",

    "STAGE16_LIGHTGBM_TUNED":
        "CPUCOND_011",

    "STAGE16_CATBOOST_TUNED":
        "CPUCOND_021",

    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING":
        "CPUCOND_031",

    "STAGE20_MASKED_CNN_V1":
        "CPUCOND_041",

    "STAGE21_MASKED_VIT_V1":
        "CPUCOND_051",
}


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def git(*args):

    p = subprocess.run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def load_csv(path):

    with Path(path).open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def show_float(
    label,
    value,
    stored,
):

    value = float(
        value
    )

    print(
        f"    {label:32s} "
        f"repr={repr(value):22s} "
        f"hex={value.hex():24s} "
        f"exact={'YES' if value == stored else 'NO'}"
    )


# =============================================================================
# 2. DURABLE / CLEAN GATE
# =============================================================================

banner(
    "STAGE26-6E-DIAG :: DURABLE STATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed after failed Stage26-6E audit."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed after failed Stage26-6E audit."
    )


if status:

    raise RuntimeError(
        "Repository is not clean."
    )


out = Path(
    "/kaggle/working/stage26_deployment_profiling/"
    "pareto/stage26_6e_bootstrap_uncertainty_provenance_audit.json"
)


print(
    "6E transient audit exists:",
    out.exists()
)


if out.exists():

    raise RuntimeError(
        "Unexpected completed 6E transient audit exists."
    )


# =============================================================================
# 3. SOURCE IDENTITY
# =============================================================================

banner(
    "STAGE26-6E-DIAG :: SOURCE IDENTITY"
)


actual_raw_sha = sha256_file(
    WARM_RAW
)

actual_summary_sha = sha256_file(
    WARM_SUMMARY
)


print(
    "Warm raw expected :",
    EXPECTED_WARM_RAW_SHA256
)

print(
    "Warm raw actual   :",
    actual_raw_sha
)

print(
    "Warm summary expected:",
    EXPECTED_WARM_SUMMARY_SHA256
)

print(
    "Warm summary actual  :",
    actual_summary_sha
)


if actual_raw_sha != EXPECTED_WARM_RAW_SHA256:

    raise RuntimeError(
        "Warm raw identity mismatch."
    )


if actual_summary_sha != EXPECTED_WARM_SUMMARY_SHA256:

    raise RuntimeError(
        "Warm summary identity mismatch."
    )


print(
    "NumPy version:",
    np.__version__
)


# =============================================================================
# 4. LOAD FROZEN RAW / SUMMARY
# =============================================================================

raw_rows = load_csv(
    WARM_RAW
)

summary_rows = load_csv(
    WARM_SUMMARY
)


# =============================================================================
# 5. EXACT P95 SEMANTICS AUDIT
# =============================================================================

banner(
    "STAGE26-6E-DIAG :: EXACT P95 COMPUTATION PATHS"
)


path_match_counts = {
    "percentile_ns_then_divide":
        0,

    "percentile_ms":
        0,

    "quantile_ns_then_divide":
        0,

    "quantile_ms":
        0,
}


per_target = {}


for target, condition_id in PRIMARY_CONDITIONS.items():

    raw_matches = [
        row
        for row in raw_rows
        if (
            row[
                "target_id"
            ]
            ==
            target
            and
            row[
                "condition_id"
            ]
            ==
            condition_id
        )
    ]


    summary_matches = [
        row
        for row in summary_rows
        if (
            row[
                "target_id"
            ]
            ==
            target
            and
            row[
                "condition_id"
            ]
            ==
            condition_id
            and
            row[
                "hardware_mode"
            ]
            ==
            "CPU_1_PHYSICAL_CORE"
            and
            int(
                row[
                    "batch_size"
                ]
            )
            ==
            1
        )
    ]


    if len(
        raw_matches
    ) != 200:

        raise RuntimeError(
            f"{target}: expected 200 raw observations."
        )


    if len(
        summary_matches
    ) != 1:

        raise RuntimeError(
            f"{target}: expected exactly one summary row."
        )


    elapsed_ns_int = np.asarray(
        [
            int(
                row[
                    "elapsed_ns"
                ]
            )
            for row in raw_matches
        ],
        dtype=np.int64,
    )


    elapsed_ns_float = elapsed_ns_int.astype(
        np.float64
    )

    elapsed_ms = (
        elapsed_ns_float
        /
        1_000_000.0
    )


    stored = float(
        summary_matches[
            0
        ][
            "p95_batch_latency_ms"
        ]
    )


    # -------------------------------------------------------------
    # Candidate A:
    # percentile in nanoseconds, divide returned scalar afterward
    # -------------------------------------------------------------

    a = (
        np.percentile(
            elapsed_ns_int,
            95,
            method="linear",
        )
        /
        1_000_000.0
    )


    # -------------------------------------------------------------
    # Candidate B:
    # convert every observation to ms first, then percentile
    # -------------------------------------------------------------

    b = np.percentile(
        elapsed_ms,
        95,
        method="linear",
    )


    # -------------------------------------------------------------
    # Candidate C:
    # quantile in ns, divide scalar afterward
    # -------------------------------------------------------------

    c = (
        np.quantile(
            elapsed_ns_int,
            0.95,
            method="linear",
        )
        /
        1_000_000.0
    )


    # -------------------------------------------------------------
    # Candidate D:
    # convert every observation to ms first, then quantile
    # -------------------------------------------------------------

    d = np.quantile(
        elapsed_ms,
        0.95,
        method="linear",
    )


    candidates = {
        "percentile_ns_then_divide":
            float(
                a
            ),

        "percentile_ms":
            float(
                b
            ),

        "quantile_ns_then_divide":
            float(
                c
            ),

        "quantile_ms":
            float(
                d
            ),
    }


    print(
        "\n",
        target,
        sep="",
    )

    print(
        "  condition:",
        condition_id
    )

    print(
        "  stored repr:",
        repr(
            stored
        )
    )

    print(
        "  stored hex :",
        stored.hex()
    )


    for name, value in candidates.items():

        show_float(
            name,
            value,
            stored,
        )


        if value == stored:

            path_match_counts[
                name
            ] += 1


    per_target[
        target
    ] = {
        "condition_id":
            condition_id,

        "stored":
            stored,

        "stored_hex":
            stored.hex(),

        "candidates": {
            name: {
                "value":
                    value,

                "hex":
                    value.hex(),

                "exact_match":
                    (
                        value == stored
                    ),
            }
            for name, value in candidates.items()
        },
    }


# =============================================================================
# 6. AGGREGATE PATH RESULT
# =============================================================================

banner(
    "STAGE26-6E-DIAG :: AGGREGATE EXACT-MATCH RESULT"
)


for name, count in path_match_counts.items():

    print(
        f"{name:32s}: "
        f"{count}/6 exact matches"
    )


exact_all_paths = [
    name
    for name, count in path_match_counts.items()
    if count == 6
]


print(
    "\nPaths reproducing all six exactly:"
)

print(
    " ",
    exact_all_paths
)


# =============================================================================
# 7. INSPECT COMMITTED HISTORICAL IMPLEMENTATION
# =============================================================================

banner(
    "STAGE26-6E-DIAG :: COMMITTED PERCENTILE IMPLEMENTATION EVIDENCE"
)


patterns = [
    "np.percentile",
    "numpy.percentile",
    "np.quantile",
    "p95_ci95_low_ms",
    "p95_batch_latency_ms",
    "bootstrap",
    "stable_seed",
]


evidence = []


for pattern in patterns:

    p = subprocess.run(
        [
            "git",
            "grep",
            "-n",
            "-I",
            "-e",
            pattern,
            "--",
            "*.py",
            "*.json",
            "*.md",
            "*.txt",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )


    if p.returncode not in {
        0,
        1,
    }:

        raise RuntimeError(
            p.stdout
        )


    for line in p.stdout.splitlines():

        lower = line.lower()


        if (
            "stage26"
            not in lower
            and
            "stage26_" not in lower
            and
            "bootstrap"
            not in lower
        ):

            continue


        evidence.append(
            line
        )


# Deduplicate.
dedup = []

seen = set()


for line in evidence:

    if line in seen:
        continue

    seen.add(
        line
    )

    dedup.append(
        line
    )


evidence = dedup


print(
    "Evidence rows:",
    len(
        evidence
    )
)


for line in evidence[
    :120
]:

    print(
        "\n",
        line,
        sep="",
    )


if len(
    evidence
) > 120:

    print(
        "\n...",
        len(
            evidence
        )
        -
        120,
        "additional rows omitted from display."
    )


# =============================================================================
# 8. EXACT DIAGNOSIS
# =============================================================================

banner(
    "STAGE26-6E-DIAG :: DIAGNOSIS"
)


if len(
    exact_all_paths
) == 1:

    diagnosis = (
        "UNAMBIGUOUS_EXACT_HISTORICAL_POINT_ESTIMATE_PATH"
    )


elif len(
    exact_all_paths
) > 1:

    diagnosis = (
        "MULTIPLE_NUMPY_PATHS_BINARY_EQUIVALENT_FOR_ALL_SIX"
    )


else:

    diagnosis = (
        "NO_TESTED_NUMPY_PATH_REPRODUCES_ALL_SIX__"
        "REQUIRES_FURTHER_NARROW_IMPLEMENTATION_AUDIT"
    )


print(
    "Diagnosis:"
)

print(
    " ",
    diagnosis
)


print(
    "\nExact-all paths:"
)

for path in exact_all_paths:

    print(
        " ",
        path
    )


print(
    "\nIMPORTANT:"
)

print(
    "  No tolerance introduced."
)

print(
    "  No stored Stage26 value changed."
)

print(
    "  No bootstrap executed."
)

print(
    "  No confidence interval regenerated."
)

print(
    "  Stage26-6D frontier unchanged."
)


# =============================================================================
# 9. FINAL DURABLE-STATE CHECK
# =============================================================================

banner(
    "STAGE26-6E-DIAG COMPLETE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed during diagnosis."
    )


if final_remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed during diagnosis."
    )


if final_status:

    raise RuntimeError(
        "Diagnostic unexpectedly modified Git."
    )


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  raw timing observations read       : YES"
)

print(
    "  stored point estimates read        : YES"
)

print(
    "  exact computation paths tested     : YES"
)

print(
    "  tolerance used                     : NO"
)

print(
    "  bootstrap executed                 : NO"
)

print(
    "  confidence intervals changed       : NO"
)

print(
    "  PR-AUC recomputed                  : NO"
)

print(
    "  holdout reopened                   : NO"
)

print(
    "  model loaded                       : NO"
)

print(
    "  inference performed                : NO"
)

print(
    "  timing performed                   : NO"
)

print(
    "  Pareto frontier changed            : NO"
)

print(
    "  GPU                                : NO"
)

print(
    "  Git modified                       : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Use the exact historical binary64 computation path identified"
)

print(
    "  here when resuming Stage26-6E."
)

print(
    "  Then continue the bootstrap-seed provenance audit."
)

print(
    "  Do not relax equality."
)


STAGE26-6E-DIAG :: DURABLE STATE
Expected parent: ff9d329785c6cd30d273729356f402e59dc4e844
Local HEAD     : ff9d329785c6cd30d273729356f402e59dc4e844
origin/main    : ff9d329785c6cd30d273729356f402e59dc4e844
Repo clean     : True
6E transient audit exists: False

STAGE26-6E-DIAG :: SOURCE IDENTITY
Warm raw expected : 78c58289ccfc4598966d6516201028f43c4b1d16d8bd0268a0cf58129d4179fa
Warm raw actual   : 78c58289ccfc4598966d6516201028f43c4b1d16d8bd0268a0cf58129d4179fa
Warm summary expected: 75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232
Warm summary actual  : 75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232
NumPy version: 2.0.2

STAGE26-6E-DIAG :: EXACT P95 COMPUTATION PATHS

STAGE16_XGBOOST_TUNED
  condition: CPUCOND_001
  stored repr: 0.5937109499999996
  stored hex : 0x1.2ffae1b30ddebp-1
    percentile_ns_then_divide        repr=0.5937109499999997     hex=0x1.2ffae1b30ddecp-1     exact=NO
    percentile_ms                    repr=0.5937109499999996     

In [19]:
# =============================================================================
# STAGE26-6E1
# NARROW READ-ONLY BOOTSTRAP SEED-PROVENANCE AUDIT
#
# DURABLE SCIENTIFIC PARENT:
#   ff9d329785c6cd30d273729356f402e59dc4e844
#
# PURPOSE
# -------
# Establish the exact committed implementation/provenance of Stage26-2
# bootstrap confidence intervals.
#
# FROZEN PROTOCOL:
#   bootstrap enabled : True
#   interval          : PERCENTILE_95_PERCENT
#   replicates        : 2000
#   RNG               : numpy.random.default_rng / PCG64
#   seed              : 26042
#
# ALREADY ESTABLISHED:
#   - Stage26-2 raw CPU1/B1 observations are intact.
#   - Existing p95 CIs exist for all six Pareto points.
#   - Historical p95 point estimates are computed in millisecond space.
#   - No tolerance is required.
#   - Stage26-6D frontier is immutable.
#
# THIS CELL DOES:
#   1. verify durable state / exact artifact identities;
#   2. inspect exact committed Stage26-2 implementation metadata;
#   3. locate every committed definition/use of:
#         stable_seed
#         bootstrap
#         default_rng
#         PCG64
#         p95_ci95_low_ms
#         p95_ci95_high_ms
#   4. print source-code context around relevant matches;
#   5. inspect the Stage26-2 commit itself for bootstrap implementation clues;
#   6. classify the EXISTING CI seed semantics.
#
# THIS CELL DOES NOT:
#   - execute bootstrap resampling
#   - regenerate confidence intervals
#   - modify Stage26-2 summaries
#   - recompute PR-AUC
#   - reopen holdout/predictions
#   - load models
#   - run inference
#   - run timing
#   - modify Pareto membership
#   - access PCAP
#   - access Release corpus
#   - use GPU
#   - modify Git
#
# OUTPUT:
#   TRANSIENT ONLY:
#
#   /kaggle/working/stage26_deployment_profiling/pareto/
#       stage26_6e1_bootstrap_seed_provenance.json
#
# If derived seeds are confirmed, the NEXT step will be a separate
# pre-correction implementation lock BEFORE any corrected CI is calculated.
# =============================================================================

from __future__ import annotations

import json
import re
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "ff9d329785c6cd30d273729356f402e59dc4e844"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

PARETO_RUNTIME = (
    STAGE26_ROOT
    / "pareto"
)

OUT = (
    PARETO_RUNTIME
    / "stage26_6e1_bootstrap_seed_provenance.json"
)


PROTOCOL = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)


WARM_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_2_cpu_warm_inference"
)

WARM_IMPLEMENTATION = (
    WARM_DIR
    / "stage26_2_warm_cpu_implementation.json"
)

WARM_RECEIPT = (
    WARM_DIR
    / "stage26_2_warm_cpu_receipt.json"
)

WARM_SUMMARY_JSON = (
    WARM_DIR
    / "stage26_2_warm_summary.json"
)

WARM_SUMMARY_CSV = (
    WARM_DIR
    / "stage26_2_warm_summary.csv"
)

WARM_RAW = (
    WARM_DIR
    / "stage26_2_warm_raw.csv"
)


EXPECTED_IMPLEMENTATION_SHA256 = (
    "678c53993799f66e1528808f46c9c27c64b8821dcb8de2ce83343f126f848d43"
)

EXPECTED_RECEIPT_SHA256 = (
    "b4b2623eabde7dd6b9acc250357a9f1ad61fa342c1dd496dcb634d4bfaca4d15"
)

EXPECTED_SUMMARY_CSV_SHA256 = (
    "75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232"
)

EXPECTED_RAW_SHA256 = (
    "78c58289ccfc4598966d6516201028f43c4b1d16d8bd0268a0cf58129d4179fa"
)


STAGE26_2_COMMIT = (
    "7ffdba3f4ca4ea5cc53097d62aaf27957009b9f6"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
    text=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            output
        )

    return p


def git(
    *args,
    check=True,
):

    return run(
        [
            "git",
            *args,
        ],
        check=check,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def recursively_find(
    obj,
    *,
    path="$",
    predicate,
    result=None,
):

    if result is None:

        result = []


    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            child = (
                path
                +
                "."
                +
                str(
                    key
                )
            )


            if predicate(
                str(
                    key
                ),
                value,
            ):

                result.append(
                    {
                        "path":
                            child,

                        "key":
                            str(
                                key
                            ),

                        "value":
                            value,
                    }
                )


            recursively_find(
                value,
                path=child,
                predicate=predicate,
                result=result,
            )


    elif isinstance(
        obj,
        list,
    ):

        for index, value in enumerate(
            obj
        ):

            recursively_find(
                value,
                path=(
                    path
                    +
                    f"[{index}]"
                ),
                predicate=predicate,
                result=result,
            )


    return result


def git_grep(
    pattern,
):

    p = run(
        [
            "git",
            "grep",
            "-n",
            "-I",
            "-e",
            pattern,
            "--",
            "*.py",
            "*.json",
            "*.md",
            "*.txt",
        ],
        check=False,
        text=True,
    )


    if p.returncode not in {
        0,
        1,
    }:

        raise RuntimeError(
            f"git grep failed for {pattern!r}:\n"
            +
            p.stdout
        )


    return [
        line
        for line in p.stdout.splitlines()
        if line.strip()
    ]


def read_context(
    relpath,
    line_number,
    radius=8,
):

    path = (
        REPO
        /
        relpath
    )


    if not path.is_file():

        return None


    try:

        lines = path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()

    except Exception:

        return None


    start = max(
        1,
        line_number
        -
        radius
    )

    end = min(
        len(
            lines
        ),
        line_number
        +
        radius
    )


    selected = []


    for n in range(
        start,
        end
        +
        1,
    ):

        selected.append(
            f"{n:6d}: {lines[n - 1]}"
        )


    return "\n".join(
        selected
    )


# =============================================================================
# 2. DURABLE STATE
# =============================================================================

banner(
    "STAGE26-6E1 :: DURABLE STATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)

print(
    "Transient output exists:",
    OUT.exists()
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-6E1 parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before seed-provenance audit."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if OUT.exists():

    raise RuntimeError(
        "Stage26-6E1 transient output already exists."
    )


# =============================================================================
# 3. EXACT ARTIFACT IDENTITY
# =============================================================================

banner(
    "STAGE26-6E1 :: EXACT ARTIFACT IDENTITY"
)


checks = [
    (
        "measurement protocol",
        PROTOCOL,
        EXPECTED_PROTOCOL_SHA256,
    ),

    (
        "warm implementation",
        WARM_IMPLEMENTATION,
        EXPECTED_IMPLEMENTATION_SHA256,
    ),

    (
        "warm receipt",
        WARM_RECEIPT,
        EXPECTED_RECEIPT_SHA256,
    ),

    (
        "warm summary CSV",
        WARM_SUMMARY_CSV,
        EXPECTED_SUMMARY_CSV_SHA256,
    ),

    (
        "warm raw CSV",
        WARM_RAW,
        EXPECTED_RAW_SHA256,
    ),
]


identity = {}


for label, path, expected in checks:

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:26s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Artifact identity mismatch: {label}"
        )


    identity[
        label
    ] = {
        "path":
            str(
                path.relative_to(
                    REPO
                )
            ),

        "sha256":
            actual,
    }


# =============================================================================
# 4. FROZEN PROTOCOL SEED
# =============================================================================

banner(
    "STAGE26-6E1 :: FROZEN BOOTSTRAP PROTOCOL"
)


protocol = json.loads(
    PROTOCOL.read_text(
        encoding="utf-8"
    )
)


bootstrap = protocol[
    "statistics_protocol"
][
    "bootstrap"
]


print(
    json.dumps(
        bootstrap,
        indent=2,
        sort_keys=True,
    )
)


if bootstrap != {
    "enabled":
        True,

    "interval":
        "PERCENTILE_95_PERCENT",

    "replicates":
        2000,

    "rng":
        "numpy.random.default_rng_PCG64",

    "seed":
        26042,
}:

    raise RuntimeError(
        "Frozen bootstrap protocol changed."
    )


print(
    "\nFrozen bootstrap seed:",
    bootstrap[
        "seed"
    ]
)


# =============================================================================
# 5. STRUCTURED IMPLEMENTATION METADATA
# =============================================================================

banner(
    "STAGE26-6E1 :: STRUCTURED STAGE26-2 IMPLEMENTATION METADATA"
)


implementation = json.loads(
    WARM_IMPLEMENTATION.read_text(
        encoding="utf-8"
    )
)

receipt = json.loads(
    WARM_RECEIPT.read_text(
        encoding="utf-8"
    )
)


predicate = lambda key, value: any(
    token in key.lower()
    for token in [
        "bootstrap",
        "seed",
        "rng",
        "percentile",
        "quantile",
        "ci95",
    ]
)


structured_evidence = {}


for label, obj in [
    (
        "implementation",
        implementation,
    ),

    (
        "receipt",
        receipt,
    ),
]:

    evidence = recursively_find(
        obj,
        predicate=predicate,
    )


    structured_evidence[
        label
    ] = evidence


    print(
        "\n",
        label.upper(),
        sep="",
    )


    if not evidence:

        print(
            "  [NO MATCHING STRUCTURED FIELDS]"
        )


    for item in evidence:

        print(
            " ",
            item[
                "path"
            ],
            "=",
            repr(
                item[
                    "value"
                ]
            ),
        )


# =============================================================================
# 6. FOCUSED REPOSITORY SEARCH
# =============================================================================

banner(
    "STAGE26-6E1 :: FOCUSED COMMITTED IMPLEMENTATION SEARCH"
)


patterns = [
    "stable_seed",
    "def bootstrap",
    "bootstrap_ci",
    "bootstrap_interval",
    "default_rng",
    "PCG64",
    "p95_ci95_low_ms",
    "p95_ci95_high_ms",
    "26042",
]


grep_rows = []


for pattern in patterns:

    rows = git_grep(
        pattern
    )


    for row in rows:

        # Keep only Stage26-relevant or executable implementation evidence.
        lower = row.lower()


        if not (
            "stage26"
            in lower
            or
            "stable_seed"
            in lower
            or
            "def bootstrap"
            in lower
            or
            "default_rng"
            in lower
            or
            "pcg64"
            in lower
        ):

            continue


        grep_rows.append(
            {
                "pattern":
                    pattern,

                "row":
                    row,
            }
        )


# Deduplicate exact grep rows.
seen = set()

deduped = []


for item in grep_rows:

    key = item[
        "row"
    ]


    if key in seen:
        continue


    seen.add(
        key
    )

    deduped.append(
        item
    )


grep_rows = deduped


print(
    "Unique relevant matches:",
    len(
        grep_rows
    )
)


# =============================================================================
# 7. PRINT HIGH-VALUE SOURCE CONTEXT
# =============================================================================

banner(
    "STAGE26-6E1 :: HIGH-VALUE SOURCE CONTEXT"
)


HIGH_VALUE_REGEX = re.compile(
    r"(?i)"
    r"(stable_seed"
    r"|def\s+bootstrap"
    r"|bootstrap_ci"
    r"|bootstrap_interval"
    r"|default_rng"
    r"|PCG64)"
)


high_value = []


for item in grep_rows:

    row = item[
        "row"
    ]


    if not HIGH_VALUE_REGEX.search(
        row
    ):

        continue


    match = re.match(
        r"^(.*?):(\d+):(.*)$",
        row,
    )


    if not match:

        continue


    relpath = match.group(
        1
    )

    line_number = int(
        match.group(
            2
        )
    )

    text = match.group(
        3
    )


    # Ignore protocol/result JSON unless no code evidence exists later.
    suffix = Path(
        relpath
    ).suffix.lower()


    evidence = {
        "path":
            relpath,

        "line":
            line_number,

        "match":
            text,

        "suffix":
            suffix,

        "context":
            read_context(
                relpath,
                line_number,
                radius=10,
            ),
    }


    high_value.append(
        evidence
    )


# Print Python first, then other sources.
high_value.sort(
    key=lambda x: (
        0
        if x[
            "suffix"
        ]
        ==
        ".py"
        else
        1,

        x[
            "path"
        ],

        x[
            "line"
        ],
    )
)


for index, item in enumerate(
    high_value[
        :80
    ],
    start=1,
):

    print(
        "\n"
        +
        "-" * 124
    )

    print(
        f"[{index}] "
        f"{item['path']}:{item['line']}"
    )

    print(
        "-" * 124
    )

    print(
        item[
            "context"
        ]
        or
        item[
            "match"
        ]
    )


if len(
    high_value
) > 80:

    print(
        "\n...",
        len(
            high_value
        )
        -
        80,
        "additional high-value matches retained in transient output."
    )


# =============================================================================
# 8. STAGE26-2 COMMIT DIFF / CONTENT SEARCH
# =============================================================================

banner(
    "STAGE26-6E1 :: STAGE26-2 COMMIT PROVENANCE"
)


commit_subject = git(
    "show",
    "-s",
    "--format=%s",
    STAGE26_2_COMMIT,
)

commit_parent = git(
    "rev-parse",
    f"{STAGE26_2_COMMIT}^",
)


print(
    "Commit :",
    STAGE26_2_COMMIT
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


changed_files = git(
    "diff-tree",
    "--no-commit-id",
    "--name-only",
    "-r",
    STAGE26_2_COMMIT,
).splitlines()


print(
    "\nChanged files:",
    len(
        changed_files
    )
)


for path in changed_files:

    print(
        " ",
        path
    )


# Search commit patch itself for seed/bootstrap semantics.
patch = git(
    "show",
    "--format=",
    "--no-ext-diff",
    STAGE26_2_COMMIT,
)


patch_lines = patch.splitlines()


patch_evidence = []


for line_number, line in enumerate(
    patch_lines,
    start=1,
):

    lower = line.lower()


    if any(
        token in lower
        for token in [
            "stable_seed",
            "bootstrap",
            "default_rng",
            "pcg64",
            "26042",
            "p95_ci95",
        ]
    ):

        patch_evidence.append(
            {
                "patch_line":
                    line_number,

                "text":
                    line[
                        :4000
                    ],
            }
        )


print(
    "\nBootstrap/seed evidence in Stage26-2 commit patch:",
    len(
        patch_evidence
    )
)


for item in patch_evidence[
    :120
]:

    print(
        f"  patch L{item['patch_line']}: "
        f"{item['text']}"
    )


if len(
    patch_evidence
) > 120:

    print(
        "  ...",
        len(
            patch_evidence
        )
        -
        120,
        "additional patch evidence rows retained."
    )


# =============================================================================
# 9. SEMANTIC CLASSIFICATION
# =============================================================================

banner(
    "STAGE26-6E1 :: SEED-PROVENANCE CLASSIFICATION"
)


all_text = "\n".join(
    item[
        "row"
    ]
    for item in grep_rows
)


structured_text = json.dumps(
    structured_evidence,
    sort_keys=True,
    default=str,
)


patch_text = "\n".join(
    item[
        "text"
    ]
    for item in patch_evidence
)


combined = (
    all_text
    +
    "\n"
    +
    structured_text
    +
    "\n"
    +
    patch_text
)


stable_seed_found = (
    re.search(
        r"\bstable_seed\b",
        combined,
        flags=re.IGNORECASE,
    )
    is not None
)


literal_rng_26042_found = (
    re.search(
        r"default_rng\s*\(\s*26042\s*\)",
        combined,
        flags=re.IGNORECASE,
    )
    is not None
)


pcg64_literal_26042_found = (
    re.search(
        r"PCG64\s*\(\s*26042\s*\)",
        combined,
        flags=re.IGNORECASE,
    )
    is not None
)


derived_seed_expression_found = (
    re.search(
        r"stable_seed\s*\([^)]*(condition|metric|condition_id)",
        combined,
        flags=re.IGNORECASE,
    )
    is not None
)


# Look for general patterns such as:
# seed = stable_seed(...)
# rng = np.random.default_rng(seed)
seed_assignment_to_stable = (
    re.search(
        r"(?:seed|bootstrap_seed)\s*=\s*stable_seed\s*\(",
        combined,
        flags=re.IGNORECASE,
    )
    is not None
)


rng_uses_variable_seed = (
    re.search(
        r"default_rng\s*\(\s*(?:seed|bootstrap_seed|rng_seed)\s*\)",
        combined,
        flags=re.IGNORECASE,
    )
    is not None
)


if (
    literal_rng_26042_found
    or
    pcg64_literal_26042_found
):

    classification = (
        "EXPLICIT_LITERAL_26042_RNG_EVIDENCE_FOUND"
    )


elif (
    derived_seed_expression_found
    or
    (
        stable_seed_found
        and
        seed_assignment_to_stable
        and
        rng_uses_variable_seed
    )
):

    classification = (
        "DERIVED_BOOTSTRAP_SEED_IMPLEMENTATION_EVIDENCE_FOUND"
    )


elif stable_seed_found:

    classification = (
        "STABLE_SEED_EVIDENCE_FOUND_BUT_EXACT_BOOTSTRAP_USE_NOT_YET_PROVEN"
    )


else:

    classification = (
        "BOOTSTRAP_SEED_IMPLEMENTATION_NOT_CERTIFIED_FROM_COMMITTED_EVIDENCE"
    )


print(
    "stable_seed evidence                  :",
    stable_seed_found
)

print(
    "derived seed expression               :",
    derived_seed_expression_found
)

print(
    "seed assigned from stable_seed        :",
    seed_assignment_to_stable
)

print(
    "default_rng(variable seed)            :",
    rng_uses_variable_seed
)

print(
    "default_rng(26042) literal            :",
    literal_rng_26042_found
)

print(
    "PCG64(26042) literal                  :",
    pcg64_literal_26042_found
)


print(
    "\nCLASSIFICATION:"
)

print(
    " ",
    classification
)


# =============================================================================
# 10. SCIENTIFIC CONSEQUENCE
# =============================================================================

if classification == (
    "EXPLICIT_LITERAL_26042_RNG_EVIDENCE_FOUND"
):

    next_action = (
        "Audit whether existing Stage26-2 latency confidence intervals "
        "numerically reproduce the frozen literal-seed implementation. "
        "Do not recompute predictive uncertainty."
    )


elif classification == (
    "DERIVED_BOOTSTRAP_SEED_IMPLEMENTATION_EVIDENCE_FOUND"
):

    next_action = (
        "Freeze a pre-correction derived-summary implementation erratum "
        "before computing any corrected timing confidence interval. "
        "Use the preserved raw observations, 2000 percentile-bootstrap "
        "replicates, NumPy default_rng/PCG64, literal seed 26042, and the "
        "historical millisecond-space statistic implementation. "
        "Do not rerun model inference."
    )


else:

    next_action = (
        "Perform one further narrow source-provenance inspection before "
        "accepting or correcting existing latency confidence intervals."
    )


print(
    "\nSCIENTIFIC CONSEQUENCE:"
)

print(
    " ",
    next_action
)


print(
    "\nPR-AUC uncertainty:"
)

print(
    "  Stage26 bootstrap target : NO"
)

print(
    "  post-hoc bootstrap       : PROHIBITED"
)


print(
    "\nStage26-6D frontier:"
)

print(
    "  immutable                : YES"
)

print(
    "  may CI correction alter it: NO"
)


# =============================================================================
# 11. WRITE TRANSIENT AUDIT ONLY
# =============================================================================

banner(
    "STAGE26-6E1 :: WRITE TRANSIENT PROVENANCE AUDIT"
)


PARETO_RUNTIME.mkdir(
    parents=True,
    exist_ok=True,
)


payload = {
    "schema":
        "stage26_6e1_bootstrap_seed_provenance_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "status":
        classification,

    "artifact_identity":
        identity,

    "stage26_2_commit": {
        "commit":
            STAGE26_2_COMMIT,

        "parent":
            commit_parent,

        "subject":
            commit_subject,

        "changed_files":
            changed_files,

        "bootstrap_seed_patch_evidence":
            patch_evidence,
    },

    "frozen_bootstrap_protocol":
        bootstrap,

    "structured_stage26_2_evidence":
        structured_evidence,

    "grep_evidence":
        grep_rows,

    "high_value_source_context":
        high_value,

    "classification_flags": {
        "stable_seed_found":
            stable_seed_found,

        "derived_seed_expression_found":
            derived_seed_expression_found,

        "seed_assignment_to_stable":
            seed_assignment_to_stable,

        "rng_uses_variable_seed":
            rng_uses_variable_seed,

        "literal_default_rng_26042_found":
            literal_rng_26042_found,

        "literal_PCG64_26042_found":
            pcg64_literal_26042_found,
    },

    "classification":
        classification,

    "scientific_consequence":
        next_action,

    "fixed_point_estimate_semantics": {
        "latency_unit_before_percentile":
            "milliseconds",

        "historical_exact_reproduction":
            (
                "Convert each elapsed_ns observation to float64 milliseconds "
                "before NumPy percentile/quantile evaluation."
            ),

        "tolerance_required":
            False,
    },

    "predictive_uncertainty": {
        "Stage26_PR_AUC_bootstrap_preregistered":
            False,

        "post_hoc_PR_AUC_bootstrap_allowed":
            False,
    },

    "scientific_state": {
        "bootstrap_executed":
            False,

        "confidence_interval_regenerated":
            False,

        "existing_confidence_interval_modified":
            False,

        "PR_AUC_recomputed":
            False,

        "holdout_reopened":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "Pareto_frontier_changed":
            False,

        "PCAP_accessed":
            False,

        "Release_corpus_accessed":
            False,

        "GPU_used":
            False,

        "Git_modified":
            False,
    },
}


with OUT.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        payload,
        f,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    )

    f.write(
        "\n"
    )


out_sha = sha256_file(
    OUT
)


print(
    "Output:"
)

print(
    " ",
    OUT
)

print(
    "SHA256:"
)

print(
    " ",
    out_sha
)


# =============================================================================
# 12. FINAL READ-ONLY AUDIT
# =============================================================================

banner(
    "STAGE26-6E1 BOOTSTRAP SEED-PROVENANCE AUDIT COMPLETE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed during Stage26-6E1."
    )


if final_remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed during Stage26-6E1."
    )


if final_status:

    raise RuntimeError(
        "Stage26-6E1 unexpectedly modified Git."
    )


print(
    "\nFINAL CLASSIFICATION:"
)

print(
    " ",
    classification
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  committed implementation inspected : YES"
)

print(
    "  Stage26-2 commit inspected          : YES"
)

print(
    "  bootstrap executed                  : NO"
)

print(
    "  CI regenerated                      : NO"
)

print(
    "  CI modified                         : NO"
)

print(
    "  PR-AUC recomputed                   : NO"
)

print(
    "  holdout reopened                    : NO"
)

print(
    "  model loaded                        : NO"
)

print(
    "  inference performed                 : NO"
)

print(
    "  timing performed                    : NO"
)

print(
    "  Pareto frontier changed             : NO"
)

print(
    "  GPU                                 : NO"
)

print(
    "  Git modified                        : NO"
)


print(
    "\nNEXT:"
)

print(
    " ",
    next_action
)


STAGE26-6E1 :: DURABLE STATE
Expected parent: ff9d329785c6cd30d273729356f402e59dc4e844
Local HEAD     : ff9d329785c6cd30d273729356f402e59dc4e844
origin/main    : ff9d329785c6cd30d273729356f402e59dc4e844
Repo clean     : True
Transient output exists: False

STAGE26-6E1 :: EXACT ARTIFACT IDENTITY
measurement protocol       PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
warm implementation        PASS 678c53993799f66e1528808f46c9c27c64b8821dcb8de2ce83343f126f848d43
warm receipt               PASS b4b2623eabde7dd6b9acc250357a9f1ad61fa342c1dd496dcb634d4bfaca4d15
warm summary CSV           PASS 75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232
warm raw CSV               PASS 78c58289ccfc4598966d6516201028f43c4b1d16d8bd0268a0cf58129d4179fa

STAGE26-6E1 :: FROZEN BOOTSTRAP PROTOCOL
{
  "enabled": true,
  "interval": "PERCENTILE_95_PERCENT",
  "replicates": 2000,
  "rng": "numpy.random.default_rng_PCG64",
  "seed": 26042
}

Frozen bootstrap seed: 26042

ST

In [20]:
# =============================================================================
# STAGE26-6E2
# FINAL READ-ONLY HISTORICAL BOOTSTRAP GENERATOR / SEED PROVENANCE AUDIT
#
# DURABLE SCIENTIFIC PARENT:
#   ff9d329785c6cd30d273729356f402e59dc4e844
#
# CONTINUITY INPUT:
#   /kaggle/working/stage26_deployment_profiling/pareto/
#       stage26_6e1_bootstrap_seed_provenance.json
#
# EXPECTED 6E1 SHA256:
#   effbbbdf6355c8b08e6eb5abdf244f1760349d54973f177fe7e779d48c1148fc
#
# PURPOSE
# -------
# Determine whether the exact Stage26-2 CI-generating bootstrap implementation
# and seed semantics are recoverable from COMMITTED provenance.
#
# IMPORTANT DISTINCTION
# ---------------------
# stage26_warm_cpu_worker.py contains RNG use for synthetic MODEL INPUT
# generation. Those RNG calls are NOT bootstrap RNG evidence.
#
# This cell therefore separates:
#
#   A. MODEL-INPUT RNG provenance
#   B. CI/BOOTSTRAP RNG provenance
#
# It searches:
#   - every text artifact committed by the Stage26-2 anchor commit;
#   - all current Stage26-2 committed files;
#   - Git history across all refs for bootstrap-generator source;
#   - commit patches introducing bootstrap-related strings;
#   - historical blobs containing relevant implementation terms.
#
# FINAL CLASSIFICATIONS
# ---------------------
# Possible outcomes:
#
#   CERTIFIED_LITERAL_SEED_26042_BOOTSTRAP_GENERATOR
#
#   CERTIFIED_DERIVED_BOOTSTRAP_SEED_GENERATOR
#
#   BOOTSTRAP_GENERATOR_FOUND_BUT_SEED_SEMANTICS_UNCERTIFIED
#
#   BOOTSTRAP_GENERATOR_NOT_RECOVERABLE_FROM_COMMITTED_PROVENANCE
#
# ABSOLUTE RULES
# --------------
#   - NO bootstrap execution.
#   - NO CI calculation.
#   - NO CI modification.
#   - NO PR-AUC calculation.
#   - NO predictive data access.
#   - NO model loading.
#   - NO inference.
#   - NO timing.
#   - NO Pareto recomputation.
#   - NO PCAP.
#   - NO Release corpus.
#   - NO GPU.
#   - NO Git modification.
#
# OUTPUT:
#   TRANSIENT ONLY:
#
#   /kaggle/working/stage26_deployment_profiling/pareto/
#       stage26_6e2_final_bootstrap_provenance_audit.json
# =============================================================================

from __future__ import annotations

import json
import re
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "ff9d329785c6cd30d273729356f402e59dc4e844"
)

STAGE26_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

PARETO_RUNTIME = (
    STAGE26_ROOT
    / "pareto"
)

SOURCE_6E1 = (
    PARETO_RUNTIME
    / "stage26_6e1_bootstrap_seed_provenance.json"
)

EXPECTED_6E1_SHA256 = (
    "effbbbdf6355c8b08e6eb5abdf244f1760349d54973f177fe7e779d48c1148fc"
)

OUT = (
    PARETO_RUNTIME
    / "stage26_6e2_final_bootstrap_provenance_audit.json"
)


STAGE26_2_COMMIT = (
    "7ffdba3f4ca4ea5cc53097d62aaf27957009b9f6"
)

EXPECTED_STAGE26_2_PARENT = (
    "46379b6d036008db4d60b056a66f4c01383e3298"
)


STAGE26_2_REL = (
    "results/stage26_deployment_profiling/"
    "stage26_2_cpu_warm_inference"
)

STAGE26_2_DIR = (
    REPO
    / STAGE26_2_REL
)


PROTOCOL = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)


WARM_RAW = (
    STAGE26_2_DIR
    / "stage26_2_warm_raw.csv"
)

EXPECTED_WARM_RAW_SHA256 = (
    "78c58289ccfc4598966d6516201028f43c4b1d16d8bd0268a0cf58129d4179fa"
)


WARM_SUMMARY = (
    STAGE26_2_DIR
    / "stage26_2_warm_summary.csv"
)

EXPECTED_WARM_SUMMARY_SHA256 = (
    "75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232"
)


WARM_IMPLEMENTATION = (
    STAGE26_2_DIR
    / "stage26_2_warm_cpu_implementation.json"
)

EXPECTED_WARM_IMPLEMENTATION_SHA256 = (
    "678c53993799f66e1528808f46c9c27c64b8821dcb8de2ce83343f126f848d43"
)


WARM_RECEIPT = (
    STAGE26_2_DIR
    / "stage26_2_warm_cpu_receipt.json"
)

EXPECTED_WARM_RECEIPT_SHA256 = (
    "b4b2623eabde7dd6b9acc250357a9f1ad61fa342c1dd496dcb634d4bfaca4d15"
)


WORKER = (
    STAGE26_2_DIR
    / "stage26_warm_cpu_worker.py"
)


# =============================================================================
# 1. SEARCH TERMS
# =============================================================================

BOOTSTRAP_STRONG_TERMS = [
    "bootstrap",
    "bootstrap_ci",
    "bootstrap_interval",
    "bootstrap_replicates",
    "bootstrap_seed",
    "resample",
    "percentile_95_percent",
    "ci95",
]


RNG_TERMS = [
    "default_rng",
    "PCG64",
    "rng",
    "seed",
]


SUMMARY_TERMS = [
    "p50_ci95_low_ms",
    "p50_ci95_high_ms",
    "p95_ci95_low_ms",
    "p95_ci95_high_ms",
    "p99_ci95_low_ms_if_n_gte_100",
    "median_throughput_ci95_low",
]


GENERATOR_SOURCE_SUFFIXES = {
    ".py",
    ".ipynb",
    ".sh",
}


TEXT_SUFFIXES = {
    ".py",
    ".json",
    ".jsonl",
    ".csv",
    ".md",
    ".txt",
    ".yaml",
    ".yml",
    ".ipynb",
    ".sh",
}


# =============================================================================
# 2. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
    text=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            output
        )

    return p


def git(
    *args,
    check=True,
):

    return run(
        [
            "git",
            *args,
        ],
        check=check,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def parse_grep_line(line):

    match = re.match(
        r"^(.*?):(\d+):(.*)$",
        line,
    )

    if not match:

        return None


    return {
        "path":
            match.group(
                1
            ),

        "line":
            int(
                match.group(
                    2
                )
            ),

        "text":
            match.group(
                3
            ),
    }


def read_context(
    relpath,
    line_number,
    radius=12,
):

    path = (
        REPO
        /
        relpath
    )


    if not path.is_file():

        return None


    try:

        lines = path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()

    except Exception:

        return None


    start = max(
        1,
        line_number
        -
        radius
    )

    end = min(
        len(
            lines
        ),
        line_number
        +
        radius
    )


    return "\n".join(
        f"{n:6d}: {lines[n - 1]}"
        for n in range(
            start,
            end
            +
            1,
        )
    )


def git_show_blob(
    commit,
    relpath,
):

    p = run(
        [
            "git",
            "show",
            f"{commit}:{relpath}",
        ],
        check=False,
        text=True,
    )


    if p.returncode != 0:

        return None


    return p.stdout


def scan_text_for_terms(
    text,
):

    lower = text.lower()


    strong = [
        term
        for term in BOOTSTRAP_STRONG_TERMS
        if term.lower() in lower
    ]


    rng = [
        term
        for term in RNG_TERMS
        if term.lower() in lower
    ]


    summary = [
        term
        for term in SUMMARY_TERMS
        if term.lower() in lower
    ]


    return {
        "bootstrap_terms":
            strong,

        "rng_terms":
            rng,

        "summary_terms":
            summary,
    }


# =============================================================================
# 3. DURABLE STATE
# =============================================================================

banner(
    "STAGE26-6E2 :: DURABLE STATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)

print(
    "6E2 output exists:",
    OUT.exists()
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-6E2 parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before final bootstrap provenance audit."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if OUT.exists():

    raise RuntimeError(
        "Stage26-6E2 transient audit already exists."
    )


# =============================================================================
# 4. CONTINUITY / SOURCE IDENTITIES
# =============================================================================

banner(
    "STAGE26-6E2 :: CONTINUITY / SOURCE IDENTITY"
)


checks = [
    (
        "Stage26-6E1",
        SOURCE_6E1,
        EXPECTED_6E1_SHA256,
    ),

    (
        "measurement protocol",
        PROTOCOL,
        EXPECTED_PROTOCOL_SHA256,
    ),

    (
        "warm raw",
        WARM_RAW,
        EXPECTED_WARM_RAW_SHA256,
    ),

    (
        "warm summary",
        WARM_SUMMARY,
        EXPECTED_WARM_SUMMARY_SHA256,
    ),

    (
        "warm implementation",
        WARM_IMPLEMENTATION,
        EXPECTED_WARM_IMPLEMENTATION_SHA256,
    ),

    (
        "warm receipt",
        WARM_RECEIPT,
        EXPECTED_WARM_RECEIPT_SHA256,
    ),
]


identity = {}


for label, path, expected in checks:

    if not path.is_file():

        raise FileNotFoundError(
            path
        )


    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:26s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Identity mismatch: {label}"
        )


    identity[
        label
    ] = actual


audit_6e1 = json.loads(
    SOURCE_6E1.read_text(
        encoding="utf-8"
    )
)


if audit_6e1[
    "classification"
] != (
    "BOOTSTRAP_SEED_IMPLEMENTATION_NOT_CERTIFIED_FROM_COMMITTED_EVIDENCE"
):

    raise RuntimeError(
        "Unexpected Stage26-6E1 classification."
    )


print(
    "\n6E1 classification:"
)

print(
    " ",
    audit_6e1[
        "classification"
    ]
)


# =============================================================================
# 5. PROTOCOL GATE
# =============================================================================

banner(
    "STAGE26-6E2 :: FROZEN STATISTICAL PROTOCOL"
)


protocol = json.loads(
    PROTOCOL.read_text(
        encoding="utf-8"
    )
)


statistics = protocol[
    "statistics_protocol"
]

bootstrap_protocol = statistics[
    "bootstrap"
]


print(
    json.dumps(
        bootstrap_protocol,
        indent=2,
        sort_keys=True,
    )
)


if bootstrap_protocol[
    "seed"
] != 26042:

    raise RuntimeError(
        "Frozen bootstrap seed changed."
    )


if bootstrap_protocol[
    "replicates"
] != 2000:

    raise RuntimeError(
        "Frozen bootstrap replicate count changed."
    )


if bootstrap_protocol[
    "rng"
] != "numpy.random.default_rng_PCG64":

    raise RuntimeError(
        "Frozen bootstrap RNG changed."
    )


if "p95_latency" not in statistics[
    "bootstrap_targets"
]:

    raise RuntimeError(
        "Frozen p95 bootstrap target disappeared."
    )


# =============================================================================
# 6. STAGE26-2 ANCHOR COMMIT IDENTITY
# =============================================================================

banner(
    "STAGE26-6E2 :: STAGE26-2 ANCHOR COMMIT"
)


commit_parent = git(
    "rev-parse",
    f"{STAGE26_2_COMMIT}^",
)

commit_subject = git(
    "show",
    "-s",
    "--format=%s",
    STAGE26_2_COMMIT,
)


print(
    "Commit :",
    STAGE26_2_COMMIT
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_STAGE26_2_PARENT:

    raise RuntimeError(
        "Stage26-2 commit parent changed."
    )


if commit_subject != (
    "stage26: anchor warm CPU deployment profiling"
):

    raise RuntimeError(
        "Unexpected Stage26-2 commit subject."
    )


# =============================================================================
# 7. INVENTORY EVERY FILE INTRODUCED BY STAGE26-2
# =============================================================================

banner(
    "STAGE26-6E2 :: STAGE26-2 COMMIT FILE-BY-FILE PROVENANCE"
)


changed_files = [
    line
    for line in git(
        "diff-tree",
        "--no-commit-id",
        "--name-only",
        "-r",
        STAGE26_2_COMMIT,
    ).splitlines()
    if line.strip()
]


stage26_2_changed = [
    relpath
    for relpath in changed_files
    if relpath.startswith(
        STAGE26_2_REL
        +
        "/"
    )
]


print(
    "Stage26-2 changed files:",
    len(
        stage26_2_changed
    )
)


commit_file_scan = []


for relpath in stage26_2_changed:

    suffix = Path(
        relpath
    ).suffix.lower()


    if suffix not in TEXT_SUFFIXES:

        continue


    text = git_show_blob(
        STAGE26_2_COMMIT,
        relpath,
    )


    if text is None:

        continue


    terms = scan_text_for_terms(
        text
    )


    if not (
        terms[
            "bootstrap_terms"
        ]
        or
        terms[
            "rng_terms"
        ]
        or
        terms[
            "summary_terms"
        ]
    ):

        continue


    record = {
        "path":
            relpath,

        "suffix":
            suffix,

        "is_executable_source":
            (
                suffix
                in
                GENERATOR_SOURCE_SUFFIXES
            ),

        **terms,
    }


    commit_file_scan.append(
        record
    )


for record in commit_file_scan:

    print(
        "\n",
        record[
            "path"
        ],
        sep="",
    )

    print(
        "  executable source :",
        record[
            "is_executable_source"
        ]
    )

    print(
        "  bootstrap terms   :",
        record[
            "bootstrap_terms"
        ]
    )

    print(
        "  rng terms         :",
        record[
            "rng_terms"
        ]
    )

    print(
        "  summary terms     :",
        record[
            "summary_terms"
        ]
    )


# =============================================================================
# 8. EXACT STAGE26-2 DIRECTORY SEARCH
# =============================================================================

banner(
    "STAGE26-6E2 :: CURRENT STAGE26-2 DIRECTORY SEARCH"
)


patterns = [
    "bootstrap",
    "default_rng",
    "PCG64",
    "26042",
    "p95_ci95_low_ms",
    "bootstrap_replicates",
]


directory_matches = []


for pattern in patterns:

    p = run(
        [
            "git",
            "grep",
            "-n",
            "-I",
            "-e",
            pattern,
            "--",
            STAGE26_2_REL,
        ],
        check=False,
        text=True,
    )


    if p.returncode not in {
        0,
        1,
    }:

        raise RuntimeError(
            p.stdout
        )


    for line in p.stdout.splitlines():

        parsed = parse_grep_line(
            line
        )


        if parsed is None:

            continue


        parsed[
            "pattern"
        ] = pattern


        directory_matches.append(
            parsed
        )


# deduplicate
seen = set()

deduped = []


for item in directory_matches:

    key = (
        item[
            "path"
        ],
        item[
            "line"
        ],
        item[
            "text"
        ],
    )


    if key in seen:

        continue


    seen.add(
        key
    )

    deduped.append(
        item
    )


directory_matches = deduped


print(
    "Unique Stage26-2 matches:",
    len(
        directory_matches
    )
)


# =============================================================================
# 9. CLASSIFY WORKER RNG CALLS AS INPUT-GENERATION OR BOOTSTRAP
# =============================================================================

banner(
    "STAGE26-6E2 :: WORKER RNG SEMANTIC CLASSIFICATION"
)


worker_text = WORKER.read_text(
    encoding="utf-8",
    errors="replace",
)


worker_lines = worker_text.splitlines()


rng_line_numbers = [
    i
    for i, line in enumerate(
        worker_lines,
        start=1,
    )
    if (
        "default_rng"
        in
        line
        or
        "PCG64"
        in
        line
    )
]


worker_rng_context = []


for line_number in rng_line_numbers:

    context = read_context(
        str(
            WORKER.relative_to(
                REPO
            )
        ),
        line_number,
        radius=18,
    )


    lower = (
        context
        or
        ""
    ).lower()


    bootstrap_semantic = (
        "bootstrap"
        in
        lower
        or
        "resample"
        in
        lower
        or
        "ci95"
        in
        lower
        or
        "percentile"
        in
        lower
    )


    input_semantic = any(
        token in lower
        for token in [
            "normal(",
            "integers(",
            "images_uint8",
            "batch_size",
            "input",
            "packet",
            "payload",
            "scaled",
            "scaler",
        ]
    )


    record = {
        "line":
            line_number,

        "context":
            context,

        "bootstrap_semantic_evidence":
            bootstrap_semantic,

        "input_generation_semantic_evidence":
            input_semantic,
    }


    worker_rng_context.append(
        record
    )


    print(
        "\n"
        +
        "-" * 124
    )

    print(
        f"default_rng context near line {line_number}"
    )

    print(
        "-" * 124
    )

    print(
        context
    )

    print(
        "\nbootstrap semantic evidence :",
        bootstrap_semantic
    )

    print(
        "input-generation evidence   :",
        input_semantic
    )


worker_has_bootstrap_rng = any(
    item[
        "bootstrap_semantic_evidence"
    ]
    for item in worker_rng_context
)


print(
    "\nWorker contains RNG tied to bootstrap semantics:",
    worker_has_bootstrap_rng
)


# =============================================================================
# 10. SEARCH CURRENT REPO FOR ACTUAL BOOTSTRAP GENERATOR SOURCE
# =============================================================================

banner(
    "STAGE26-6E2 :: CURRENT COMMITTED BOOTSTRAP GENERATOR CANDIDATES"
)


candidate_source_rows = []


for pattern in [
    "def bootstrap",
    "bootstrap_ci",
    "bootstrap_interval",
    "bootstrap_replicates",
    "np.random.default_rng",
]:

    p = run(
        [
            "git",
            "grep",
            "-n",
            "-I",
            "-e",
            pattern,
            "--",
            "*.py",
            "*.ipynb",
            "*.sh",
        ],
        check=False,
        text=True,
    )


    if p.returncode not in {
        0,
        1,
    }:

        raise RuntimeError(
            p.stdout
        )


    for line in p.stdout.splitlines():

        parsed = parse_grep_line(
            line
        )


        if parsed is None:

            continue


        parsed[
            "pattern"
        ] = pattern


        candidate_source_rows.append(
            parsed
        )


# deduplicate
seen = set()

candidate_source_dedup = []


for item in candidate_source_rows:

    key = (
        item[
            "path"
        ],
        item[
            "line"
        ],
        item[
            "text"
        ],
    )


    if key in seen:

        continue


    seen.add(
        key
    )

    candidate_source_dedup.append(
        item
    )


candidate_source_rows = candidate_source_dedup


# Retain Stage26-specific implementation candidates separately.
stage26_generator_candidates = [
    item
    for item in candidate_source_rows
    if "stage26" in item[
        "path"
    ].lower()
]


print(
    "Current executable bootstrap/RNG source matches:",
    len(
        candidate_source_rows
    )
)

print(
    "Stage26 executable candidates:",
    len(
        stage26_generator_candidates
    )
)


for item in stage26_generator_candidates:

    print(
        f"\n{item['path']}:{item['line']}:"
        f"{item['text']}"
    )


# =============================================================================
# 11. SEARCH GIT HISTORY FOR BOOTSTRAP GENERATOR INTRODUCTIONS
# =============================================================================

banner(
    "STAGE26-6E2 :: ALL-REF GIT HISTORY SEARCH"
)


history_searches = {
    "bootstrap_replicates":
        "bootstrap_replicates",

    "bootstrap_seed":
        "bootstrap_seed",

    "def_bootstrap":
        "def bootstrap",

    "stable_seed":
        "stable_seed",

    "p95_ci95_low_ms":
        "p95_ci95_low_ms",
}


history_evidence = {}


for label, term in history_searches.items():

    p = run(
        [
            "git",
            "log",
            "--all",
            "--format=%H%x09%s",
            "-S",
            term,
        ],
        check=False,
        text=True,
    )


    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )


    rows = [
        line
        for line in p.stdout.splitlines()
        if line.strip()
    ]


    history_evidence[
        label
    ] = rows


    print(
        "\n",
        label,
        ":",
        len(
            rows
        ),
        "commit(s)",
        sep="",
    )


    for row in rows[
        :30
    ]:

        print(
            " ",
            row
        )


# Regex history search coupling bootstrap with 26042.
regex_history = run(
    [
        "git",
        "log",
        "--all",
        "--format=%H%x09%s",
        "-G",
        r"(bootstrap.*26042|26042.*bootstrap|default_rng\(.*26042)",
    ],
    check=False,
    text=True,
)


if regex_history.returncode != 0:

    raise RuntimeError(
        regex_history.stdout
    )


regex_history_rows = [
    line
    for line in regex_history.stdout.splitlines()
    if line.strip()
]


print(
    "\nBootstrap/26042 regex-history commits:",
    len(
        regex_history_rows
    )
)


for row in regex_history_rows[
    :50
]:

    print(
        " ",
        row
    )


# =============================================================================
# 12. INSPECT HISTORICAL COMMITS THAT TOUCHED BOOTSTRAP_REPLICATES
# =============================================================================

banner(
    "STAGE26-6E2 :: HISTORICAL BOOTSTRAP-REPLICATE COMMIT INSPECTION"
)


bootstrap_commits = []


for row in history_evidence[
    "bootstrap_replicates"
]:

    commit = row.split(
        "\t",
        1,
    )[
        0
    ]


    if commit not in bootstrap_commits:

        bootstrap_commits.append(
            commit
        )


historical_commit_inspections = []


for commit in bootstrap_commits[
    :20
]:

    subject = git(
        "show",
        "-s",
        "--format=%s",
        commit,
    )


    changed = git(
        "diff-tree",
        "--no-commit-id",
        "--name-only",
        "-r",
        commit,
    ).splitlines()


    executable_changed = [
        path
        for path in changed
        if Path(
            path
        ).suffix.lower()
        in
        GENERATOR_SOURCE_SUFFIXES
    ]


    patch_result = run(
        [
            "git",
            "show",
            "--format=",
            "--no-ext-diff",
            commit,
        ],
        check=True,
        text=True,
    )


    patch_lines = patch_result.stdout.splitlines()


    relevant_patch = []


    for line_number, line in enumerate(
        patch_lines,
        start=1,
    ):

        lower = line.lower()


        if any(
            token in lower
            for token in [
                "bootstrap",
                "26042",
                "default_rng",
                "pcg64",
                "ci95",
            ]
        ):

            relevant_patch.append(
                {
                    "line":
                        line_number,

                    "text":
                        line[
                            :4000
                        ],
                }
            )


    record = {
        "commit":
            commit,

        "subject":
            subject,

        "changed_files":
            changed,

        "executable_changed_files":
            executable_changed,

        "relevant_patch_evidence":
            relevant_patch,
    }


    historical_commit_inspections.append(
        record
    )


    print(
        "\n"
        +
        "-" * 124
    )

    print(
        commit,
        subject
    )

    print(
        "Executable files changed:",
        executable_changed
    )

    print(
        "Relevant patch rows:",
        len(
            relevant_patch
        )
    )


    for item in relevant_patch[
        :40
    ]:

        print(
            f"  L{item['line']}: "
            f"{item['text']}"
        )


# =============================================================================
# 13. STAGE26-2 EXACT PATCH — WAS A GENERATOR SOURCE COMMITTED?
# =============================================================================

banner(
    "STAGE26-6E2 :: STAGE26-2 GENERATOR-SOURCE AUDIT"
)


stage26_2_executable_changed = [
    path
    for path in stage26_2_changed
    if Path(
        path
    ).suffix.lower()
    in
    GENERATOR_SOURCE_SUFFIXES
]


print(
    "Executable files committed by Stage26-2:"
)


for path in stage26_2_executable_changed:

    print(
        " ",
        path
    )


stage26_2_generator_source_candidates = []


for relpath in stage26_2_executable_changed:

    text = git_show_blob(
        STAGE26_2_COMMIT,
        relpath,
    )


    if text is None:

        continue


    lower = text.lower()


    bootstrap_hits = [
        term
        for term in BOOTSTRAP_STRONG_TERMS
        if term.lower() in lower
    ]


    summary_hits = [
        term
        for term in SUMMARY_TERMS
        if term.lower() in lower
    ]


    rng_hits = [
        term
        for term in RNG_TERMS
        if term.lower() in lower
    ]


    record = {
        "path":
            relpath,

        "bootstrap_hits":
            bootstrap_hits,

        "summary_hits":
            summary_hits,

        "rng_hits":
            rng_hits,
    }


    stage26_2_generator_source_candidates.append(
        record
    )


    print(
        "\n",
        relpath,
        sep="",
    )

    print(
        "  bootstrap hits:",
        bootstrap_hits
    )

    print(
        "  summary hits  :",
        summary_hits
    )

    print(
        "  RNG hits      :",
        rng_hits
    )


actual_stage26_2_ci_generator_files = [
    record
    for record in stage26_2_generator_source_candidates
    if (
        record[
            "bootstrap_hits"
        ]
        and
        record[
            "summary_hits"
        ]
    )
]


print(
    "\nExecutable files containing BOTH bootstrap and CI-summary semantics:",
    len(
        actual_stage26_2_ci_generator_files
    )
)


for record in actual_stage26_2_ci_generator_files:

    print(
        " ",
        record[
            "path"
        ]
    )


# =============================================================================
# 14. COMMITTED METADATA SEED CERTIFICATION
# =============================================================================

banner(
    "STAGE26-6E2 :: COMMITTED METADATA SEED CERTIFICATION"
)


all_stage26_2_text = []


for relpath in stage26_2_changed:

    suffix = Path(
        relpath
    ).suffix.lower()


    if suffix not in TEXT_SUFFIXES:

        continue


    text = git_show_blob(
        STAGE26_2_COMMIT,
        relpath,
    )


    if text is not None:

        all_stage26_2_text.append(
            (
                relpath,
                text,
            )
        )


bootstrap_seed_metadata_evidence = []


for relpath, text in all_stage26_2_text:

    lines = text.splitlines()


    for line_number, line in enumerate(
        lines,
        start=1,
    ):

        lower = line.lower()


        # Require bootstrap/CI semantic proximity rather than any random seed.
        if (
            "26042"
            in
            line
            and
            any(
                term in lower
                for term in [
                    "bootstrap",
                    "ci",
                    "confidence",
                    "percentile",
                    "resampl",
                ]
            )
        ):

            bootstrap_seed_metadata_evidence.append(
                {
                    "path":
                        relpath,

                    "line":
                        line_number,

                    "text":
                        line,
                }
            )


        if (
            re.search(
                r"bootstrap[_\s-]*seed",
                lower,
            )
            or
            re.search(
                r"(confidence|ci)[_\s-]*seed",
                lower,
            )
        ):

            bootstrap_seed_metadata_evidence.append(
                {
                    "path":
                        relpath,

                    "line":
                        line_number,

                    "text":
                        line,
                }
            )


# deduplicate
seen = set()

metadata_dedup = []


for item in bootstrap_seed_metadata_evidence:

    key = (
        item[
            "path"
        ],
        item[
            "line"
        ],
        item[
            "text"
        ],
    )


    if key in seen:

        continue


    seen.add(
        key
    )

    metadata_dedup.append(
        item
    )


bootstrap_seed_metadata_evidence = metadata_dedup


print(
    "Stage26-2 committed bootstrap-seed metadata evidence:",
    len(
        bootstrap_seed_metadata_evidence
    )
)


for item in bootstrap_seed_metadata_evidence:

    print(
        f"  {item['path']}:{item['line']}: "
        f"{item['text']}"
    )


# =============================================================================
# 15. FINAL CLASSIFICATION
# =============================================================================

banner(
    "STAGE26-6E2 :: FINAL PROVENANCE CLASSIFICATION"
)


# -------------------------------------------------------------------------
# Evidence rules
# -------------------------------------------------------------------------

# A true CI generator must contain bootstrap/resampling semantics AND
# CI/statistic output semantics in executable source.
generator_found = bool(
    actual_stage26_2_ci_generator_files
)


# Literal certification requires an actual bootstrap generator or explicit
# bootstrap-specific committed metadata tying the CI RNG to 26042.
literal_seed_certified = False

derived_seed_certified = False

generator_seed_uncertified = False


if generator_found:

    for record in actual_stage26_2_ci_generator_files:

        text = git_show_blob(
            STAGE26_2_COMMIT,
            record[
                "path"
            ],
        )


        if text is None:

            continue


        # literal seed directly used by bootstrap RNG
        if (
            re.search(
                r"default_rng\s*\(\s*26042\s*\)",
                text,
                flags=re.IGNORECASE,
            )
            or
            re.search(
                r"PCG64\s*\(\s*26042\s*\)",
                text,
                flags=re.IGNORECASE,
            )
        ):

            literal_seed_certified = True


        # derived seed tied to bootstrap
        if (
            re.search(
                r"bootstrap.{0,300}(derived_seed|stable_seed)",
                text,
                flags=(
                    re.IGNORECASE
                    |
                    re.DOTALL
                ),
            )
            or
            re.search(
                r"(derived_seed|stable_seed).{0,300}bootstrap",
                text,
                flags=(
                    re.IGNORECASE
                    |
                    re.DOTALL
                ),
            )
        ):

            derived_seed_certified = True


    if (
        not literal_seed_certified
        and
        not derived_seed_certified
    ):

        generator_seed_uncertified = True


# Explicit committed bootstrap-specific seed metadata may independently
# certify literal 26042 even if orchestration source was not preserved.
if any(
    "26042"
    in
    item[
        "text"
    ]
    for item in bootstrap_seed_metadata_evidence
):

    literal_seed_certified = True


if literal_seed_certified:

    classification = (
        "CERTIFIED_LITERAL_SEED_26042_BOOTSTRAP_GENERATOR"
    )


elif derived_seed_certified:

    classification = (
        "CERTIFIED_DERIVED_BOOTSTRAP_SEED_GENERATOR"
    )


elif generator_seed_uncertified:

    classification = (
        "BOOTSTRAP_GENERATOR_FOUND_BUT_SEED_SEMANTICS_UNCERTIFIED"
    )


else:

    classification = (
        "BOOTSTRAP_GENERATOR_NOT_RECOVERABLE_FROM_COMMITTED_PROVENANCE"
    )


print(
    "Stage26-2 CI generator source found :",
    generator_found
)

print(
    "Literal seed 26042 certified        :",
    literal_seed_certified
)

print(
    "Derived bootstrap seed certified    :",
    derived_seed_certified
)

print(
    "Bootstrap-specific seed metadata     :",
    len(
        bootstrap_seed_metadata_evidence
    )
)


print(
    "\nFINAL CLASSIFICATION:"
)

print(
    " ",
    classification
)


# =============================================================================
# 16. SCIENTIFIC CONSEQUENCE
# =============================================================================

if classification == (
    "CERTIFIED_LITERAL_SEED_26042_BOOTSTRAP_GENERATOR"
):

    next_action = (
        "Freeze a numerical-reproduction audit protocol, then verify the "
        "stored Stage26-2 latency CIs against the certified historical "
        "implementation. No model remeasurement is needed."
    )


elif classification == (
    "CERTIFIED_DERIVED_BOOTSTRAP_SEED_GENERATOR"
):

    next_action = (
        "Freeze a derived-summary-only statistical erratum before calculating "
        "replacement timing CIs. The correction must use the preserved raw "
        "observations, 2000 percentile-bootstrap replicates, "
        "numpy.random.default_rng/PCG64 with literal seed 26042, and the "
        "historical millisecond-space statistic semantics. Existing Stage26-2 "
        "raw measurements and point estimates remain immutable."
    )


elif classification in {
    "BOOTSTRAP_GENERATOR_FOUND_BUT_SEED_SEMANTICS_UNCERTIFIED",
    "BOOTSTRAP_GENERATOR_NOT_RECOVERABLE_FROM_COMMITTED_PROVENANCE",
}:

    next_action = (
        "Treat existing Stage26-2 bootstrap CIs as NOT PROTOCOL-CERTIFIED. "
        "Before calculating any replacement interval, freeze a "
        "derived-summary-only statistical correction protocol anchored to "
        "the current scientific parent. The correction may consume only the "
        "immutable Stage26-2 raw timing observations and must implement the "
        "already-frozen protocol exactly: 2000 percentile-bootstrap "
        "replicates, numpy.random.default_rng/PCG64, literal seed 26042, "
        "with statistics evaluated in millisecond space. No inference, "
        "timing remeasurement, model loading, or predictive bootstrap."
    )


else:

    raise RuntimeError(
        "Unhandled classification."
    )


print(
    "\nSCIENTIFIC CONSEQUENCE:"
)

print(
    " ",
    next_action
)


print(
    "\nUNCHANGED SCIENCE:"
)

print(
    "  raw Stage26-2 measurements       : IMMUTABLE"
)

print(
    "  Stage26-6D Pareto membership     : IMMUTABLE"
)

print(
    "  predictive PR-AUC points         : IMMUTABLE"
)

print(
    "  PR-AUC bootstrap                 : NOT PREREGISTERED"
)

print(
    "  new model measurement required   : NO"
)


# =============================================================================
# 17. WRITE TRANSIENT AUDIT
# =============================================================================

banner(
    "STAGE26-6E2 :: WRITE TRANSIENT FINAL PROVENANCE AUDIT"
)


PARETO_RUNTIME.mkdir(
    parents=True,
    exist_ok=True,
)


payload = {
    "schema":
        "stage26_6e2_final_bootstrap_provenance_audit_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "continuity": {
        "stage26_6e1_path":
            str(
                SOURCE_6E1
            ),

        "stage26_6e1_sha256":
            EXPECTED_6E1_SHA256,

        "stage26_6e1_classification":
            audit_6e1[
                "classification"
            ],
    },

    "artifact_identity":
        identity,

    "frozen_bootstrap_protocol":
        bootstrap_protocol,

    "stage26_2_anchor": {
        "commit":
            STAGE26_2_COMMIT,

        "parent":
            commit_parent,

        "subject":
            commit_subject,

        "changed_file_count":
            len(
                stage26_2_changed
            ),

        "executable_changed_files":
            stage26_2_executable_changed,
    },

    "stage26_2_commit_file_scan":
        commit_file_scan,

    "stage26_2_directory_matches":
        directory_matches,

    "worker_rng_semantics":
        worker_rng_context,

    "worker_has_bootstrap_rng":
        worker_has_bootstrap_rng,

    "current_stage26_executable_generator_candidates":
        stage26_generator_candidates,

    "git_history": {
        "string_history":
            history_evidence,

        "bootstrap_26042_regex_history":
            regex_history_rows,

        "historical_commit_inspections":
            historical_commit_inspections,
    },

    "stage26_2_generator_source_candidates":
        stage26_2_generator_source_candidates,

    "actual_stage26_2_ci_generator_files":
        actual_stage26_2_ci_generator_files,

    "bootstrap_seed_metadata_evidence":
        bootstrap_seed_metadata_evidence,

    "classification_evidence": {
        "generator_found":
            generator_found,

        "literal_seed_26042_certified":
            literal_seed_certified,

        "derived_bootstrap_seed_certified":
            derived_seed_certified,

        "bootstrap_specific_seed_metadata_count":
            len(
                bootstrap_seed_metadata_evidence
            ),
    },

    "classification":
        classification,

    "scientific_consequence":
        next_action,

    "fixed_statistic_semantics": {
        "measurement_source":
            "IMMUTABLE_STAGE26_2_WARM_RAW",

        "statistic_unit":
            "milliseconds",

        "point_estimate_semantics":
            (
                "Convert elapsed_ns observations to float64 milliseconds "
                "before numpy percentile/quantile evaluation."
            ),

        "tolerance_required":
            False,
    },

    "scientific_state": {
        "bootstrap_executed":
            False,

        "confidence_interval_computed":
            False,

        "confidence_interval_modified":
            False,

        "predictive_metric_recomputed":
            False,

        "PR_AUC_bootstrap_computed":
            False,

        "holdout_reopened":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "Pareto_frontier_recomputed":
            False,

        "cross_group_frontier_computed":
            False,

        "PCAP_accessed":
            False,

        "Release_corpus_accessed":
            False,

        "GPU_used":
            False,

        "Git_modified":
            False,
    },
}


with OUT.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        payload,
        f,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    )

    f.write(
        "\n"
    )


out_sha = sha256_file(
    OUT
)


print(
    "Output:"
)

print(
    " ",
    OUT
)

print(
    "SHA256:"
)

print(
    " ",
    out_sha
)


# =============================================================================
# 18. FINAL READ-ONLY CLOSURE
# =============================================================================

banner(
    "STAGE26-6E2 FINAL BOOTSTRAP PROVENANCE AUDIT COMPLETE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed during Stage26-6E2."
    )


if final_remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed during Stage26-6E2."
    )


if final_status:

    raise RuntimeError(
        "Stage26-6E2 unexpectedly modified Git."
    )


print(
    "\nFINAL CLASSIFICATION:"
)

print(
    " ",
    classification
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  all committed Stage26-2 provenance audited : YES"
)

print(
    "  Git history searched                       : YES"
)

print(
    "  worker input RNG separated from bootstrap  : YES"
)

print(
    "  bootstrap executed                         : NO"
)

print(
    "  CI calculated                              : NO"
)

print(
    "  CI modified                                : NO"
)

print(
    "  PR-AUC recomputed                          : NO"
)

print(
    "  predictive bootstrap                       : NO"
)

print(
    "  holdout reopened                           : NO"
)

print(
    "  model loaded                               : NO"
)

print(
    "  inference performed                        : NO"
)

print(
    "  timing performed                           : NO"
)

print(
    "  Pareto frontier changed                    : NO"
)

print(
    "  GPU                                        : NO"
)

print(
    "  Git modified                               : NO"
)


print(
    "\nNEXT:"
)

print(
    " ",
    next_action
)


STAGE26-6E2 :: DURABLE STATE
Expected parent: ff9d329785c6cd30d273729356f402e59dc4e844
Local HEAD     : ff9d329785c6cd30d273729356f402e59dc4e844
origin/main    : ff9d329785c6cd30d273729356f402e59dc4e844
Repo clean     : True
6E2 output exists: False

STAGE26-6E2 :: CONTINUITY / SOURCE IDENTITY
Stage26-6E1                PASS effbbbdf6355c8b08e6eb5abdf244f1760349d54973f177fe7e779d48c1148fc
measurement protocol       PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
warm raw                   PASS 78c58289ccfc4598966d6516201028f43c4b1d16d8bd0268a0cf58129d4179fa
warm summary               PASS 75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232
warm implementation        PASS 678c53993799f66e1528808f46c9c27c64b8821dcb8de2ce83343f126f848d43
warm receipt               PASS b4b2623eabde7dd6b9acc250357a9f1ad61fa342c1dd496dcb634d4bfaca4d15

6E1 classification:
  BOOTSTRAP_SEED_IMPLEMENTATION_NOT_CERTIFIED_FROM_COMMITTED_EVIDENCE

STAGE26-6E2 :: FROZEN STATISTI

In [21]:
# =============================================================================
# STAGE26-6F0
# FREEZE DERIVED-SUMMARY-ONLY BOOTSTRAP CORRECTION PROTOCOL + IMPLEMENTATION
# COMMIT + PUSH + REMOTE VERIFY
#
# DURABLE SCIENTIFIC PARENT:
#   ff9d329785c6cd30d273729356f402e59dc4e844
#
# WHY THIS CHECKPOINT EXISTS
# --------------------------
# Stage26-6E1 / 6E2 established:
#
#   - Stage26-2 raw timing observations are intact.
#   - Stage26-2 point estimates are intact.
#   - Stage26-2 historical bootstrap CI values exist.
#   - the historical CI-generating source was NOT committed.
#   - no bootstrap-specific seed metadata was committed.
#   - Stage26-2 worker RNG calls belong to model-input generation,
#     NOT bootstrap resampling.
#
# FINAL 6E2 CLASSIFICATION:
#
#   BOOTSTRAP_GENERATOR_NOT_RECOVERABLE_FROM_COMMITTED_PROVENANCE
#
# Therefore:
#
#   Existing Stage26-2 CIs remain historical artifacts but are NOT
#   protocol-certified for final publication use.
#
# THIS CHECKPOINT FREEZES THE CORRECTION BEFORE ANY REPLACEMENT CI EXISTS.
#
# IMPORTANT:
# ----------
# THIS CELL DOES NOT EXECUTE THE CORRECTION WORKER.
#
# It writes and freezes:
#
#   1. exact correction protocol
#   2. exact correction worker source
#   3. Stage26-6E1 / 6E2 provenance copies
#   4. provenance/erratum record
#   5. freeze receipt
#   6. package manifest
#
# CORRECTION SCOPE
# ----------------
# ALL Stage26-2 PASS timing conditions, not merely the six Pareto points.
#
# This avoids leaving uncertified historical timing CIs elsewhere in the
# Stage26-2 publication package.
#
# ORIGINAL STAGE26-2 FILES WILL NEVER BE OVERWRITTEN.
#
# FROZEN CORRECTION SEMANTICS
# ---------------------------
#
# Input:
#   immutable stage26_2_warm_raw.csv
#
# Point estimates:
#   immutable / not replaced
#
# Bootstrap:
#   2,000 replicates
#   np.random.default_rng(26042)
#   PCG64
#   resampling WITH replacement
#
# CONDITION-LOCAL RNG:
#   For each PASS condition:
#
#       rng = np.random.default_rng(26042)
#
#   No derived seeds.
#
#   This makes seed semantics independent of condition execution order,
#   failed conditions, target identity, metric identity, or batch size.
#
# WITHIN-CONDITION RESAMPLE PAIRING:
#   Generate one B x n integer resample-index matrix.
#   Reuse that SAME matrix for all frozen timing statistics in the condition.
#
# Latency statistic space:
#
#       elapsed_ms = float64(elapsed_ns) / 1_000_000
#
#   Conversion occurs BEFORE percentile evaluation.
#
# Frozen bootstrap targets:
#   p50 latency
#   p95 latency
#   p99 latency only when n >= 100
#   median throughput
#
# Throughput:
#   bootstrap the preserved raw flows_per_second observations directly.
#
# Bootstrap CI:
#   2.5th and 97.5th percentiles of 2,000 bootstrap statistics
#   using NumPy method="linear".
#
# Maximum:
#   descriptive only, never bootstrapped.
#
# OOM / timeout:
#   no bootstrap; status remains unchanged.
#
# PARETO:
#   Stage26-6D point-estimate frontier is immutable.
#   corrected CIs cannot alter its membership.
#
# PREDICTIVE UNCERTAINTY:
#   PR-AUC bootstrap is NOT a Stage26 bootstrap target.
#   It is forbidden here.
#
# NO:
#   - bootstrap execution
#   - corrected CI calculation
#   - new#   - bootstrap execution
#   - corrected CI calculation
#   - new timing
#   - model loading
#   - inference
#   - predictive metric recomputation
#   - holdout access
#   - Pareto recomputation
#   - PCAP
#   - Release corpus
#   - GPU
# =============================================================================

from __future__ import annotations

import os
import json
import stat
import shutil
import hashlib
import subprocess
import tempfile
import textwrap
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "ff9d329785c6cd30d273729356f402e59dc4e844"
)

COMMIT_SUBJECT = (
    "stage26: freeze bootstrap CI correction protocol"
)


# -----------------------------------------------------------------------------
# Immutable Stage26 protocol
# -----------------------------------------------------------------------------

PROTOCOL = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)


# -----------------------------------------------------------------------------
# Immutable Stage26-2 inputs
# -----------------------------------------------------------------------------

WARM_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_2_cpu_warm_inference"
)

WARM_RAW = (
    WARM_DIR
    / "stage26_2_warm_raw.csv"
)

WARM_SUMMARY = (
    WARM_DIR
    / "stage26_2_warm_summary.csv"
)

WARM_STATUS = (
    WARM_DIR
    / "stage26_2_condition_status.csv"
)

WARM_RECEIPT = (
    WARM_DIR
    / "stage26_2_warm_cpu_receipt.json"
)


EXPECTED_WARM_RAW_SHA256 = (
    "78c58289ccfc4598966d6516201028f43c4b1d16d8bd0268a0cf58129d4179fa"
)

EXPECTED_WARM_SUMMARY_SHA256 = (
    "75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232"
)

EXPECTED_WARM_STATUS_SHA256 = (
    "df146563826992cb57702e589e4cf5d875ecfdbbf79533a731405e8eb738e7af"
)

EXPECTED_WARM_RECEIPT_SHA256 = (
    "b4b2623eabde7dd6b9acc250357a9f1ad61fa342c1dd496dcb634d4bfaca4d15"
)


# -----------------------------------------------------------------------------
# Immutable Stage26-6D point-estimate Pareto result
# -----------------------------------------------------------------------------

DIR_6D = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_6d_cpu_pareto_point_estimate"
)

FRONTIER_6D = (
    DIR_6D
    / "stage26_6d_point_estimate_frontiers.json"
)

RECEIPT_6D = (
    DIR_6D
    / "stage26_6d_pareto_point_estimate_receipt.json"
)

EXPECTED_FRONTIER_6D_SHA256 = (
    "367b3d34ddb4dd125dcbe3db71291b5576b4521640cd2f11d76e50dc247fc6b5"
)

EXPECTED_RECEIPT_6D_SHA256 = (
    "d898f08dc45cdb625efa5c000d76c4414a245d2091dc92274300f2260956bc79"
)


# -----------------------------------------------------------------------------
# Transient Stage26-6E provenance evidence
# -----------------------------------------------------------------------------

RUNTIME_PARETO = Path(
    "/kaggle/working/stage26_deployment_profiling/pareto"
)

SOURCE_6E1 = (
    RUNTIME_PARETO
    / "stage26_6e1_bootstrap_seed_provenance.json"
)

SOURCE_6E2 = (
    RUNTIME_PARETO
    / "stage26_6e2_final_bootstrap_provenance_audit.json"
)


EXPECTED_6E1_SHA256 = (
    "effbbbdf6355c8b08e6eb5abdf244f1760349d54973f177fe7e779d48c1148fc"
)

EXPECTED_6E2_SHA256 = (
    "3492fa1bf3cbdf7fd1e29079fccf8cfd8edaf8a9cc060a510ef5c5371bacea08"
)


# -----------------------------------------------------------------------------
# New durable checkpoint
# -----------------------------------------------------------------------------

CHECKPOINT_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_6f0_bootstrap_correction_protocol_lock"
)

CHECKPOINT_DIR = (
    REPO
    / CHECKPOINT_REL
)

PROVENANCE_DIR = (
    CHECKPOINT_DIR
    / "provenance"
)

CORRECTION_PROTOCOL = (
    CHECKPOINT_DIR
    / "stage26_6f0_bootstrap_correction_protocol.json"
)

CORRECTION_WORKER = (
    CHECKPOINT_DIR
    / "stage26_6f0_bootstrap_correction_worker.py"
)

ERRATUM_RECORD = (
    CHECKPOINT_DIR
    / "stage26_6f0_statistical_erratum_record.json"
)

FREEZE_RECEIPT = (
    CHECKPOINT_DIR
    / "stage26_6f0_bootstrap_correction_freeze_receipt.json"
)

MANIFEST = (
    CHECKPOINT_DIR
    / "stage26_6f0_bootstrap_correction_lock_manifest.json"
)

DURABLE_6E1 = (
    PROVENANCE_DIR
    / "stage26_6e1_bootstrap_seed_provenance.json"
)

DURABLE_6E2 = (
    PROVENANCE_DIR
    / "stage26_6e2_final_bootstrap_provenance_audit.json"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_json(
    path,
    payload,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        +
        ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def atomic_text(
    path,
    text,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        +
        ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as f:

        f.write(
            text
        )

        if not text.endswith(
            "\n"
        ):

            f.write(
                "\n"
            )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        text=False,
    )

    return bytes(
        p.stdout
    )


# =============================================================================
# 2. DURABLE GIT GATE
# =============================================================================

banner(
    "STAGE26-6F0 :: DURABLE SCIENTIFIC PARENT"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-6F0 parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before correction-protocol freeze."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if CHECKPOINT_DIR.exists():

    raise RuntimeError(
        "Stage26-6F0 checkpoint already exists."
    )


# =============================================================================
# 3. IMMUTABLE SOURCE IDENTITY
# =============================================================================

banner(
    "STAGE26-6F0 :: IMMUTABLE SOURCE IDENTITY"
)


checks = [
    (
        "measurement protocol",
        PROTOCOL,
        EXPECTED_PROTOCOL_SHA256,
    ),

    (
        "Stage26-2 warm raw",
        WARM_RAW,
        EXPECTED_WARM_RAW_SHA256,
    ),

    (
        "Stage26-2 historical summary",
        WARM_SUMMARY,
        EXPECTED_WARM_SUMMARY_SHA256,
    ),

    (
        "Stage26-2 condition status",
        WARM_STATUS,
        EXPECTED_WARM_STATUS_SHA256,
    ),

    (
        "Stage26-2 receipt",
        WARM_RECEIPT,
        EXPECTED_WARM_RECEIPT_SHA256,
    ),

    (
        "Stage26-6D frontier",
        FRONTIER_6D,
        EXPECTED_FRONTIER_6D_SHA256,
    ),

    (
        "Stage26-6D receipt",
        RECEIPT_6D,
        EXPECTED_RECEIPT_6D_SHA256,
    ),

    (
        "Stage26-6E1 provenance",
        SOURCE_6E1,
        EXPECTED_6E1_SHA256,
    ),

    (
        "Stage26-6E2 provenance",
        SOURCE_6E2,
        EXPECTED_6E2_SHA256,
    ),
]


identity = {}


for label, path, expected in checks:

    if not path.is_file():

        raise FileNotFoundError(
            path
        )


    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:34s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Source identity mismatch: {label}"
        )


    identity[
        label
    ] = {
        "path":
            str(
                path
            ),

        "sha256":
            actual,
    }


# =============================================================================
# 4. PROVENANCE CLASSIFICATION GATE
# =============================================================================

banner(
    "STAGE26-6F0 :: PROVENANCE CLASSIFICATION GATE"
)


audit_6e1 = json.loads(
    SOURCE_6E1.read_text(
        encoding="utf-8"
    )
)

audit_6e2 = json.loads(
    SOURCE_6E2.read_text(
        encoding="utf-8"
    )
)


expected_6e1_classification = (
    "BOOTSTRAP_SEED_IMPLEMENTATION_NOT_CERTIFIED_FROM_COMMITTED_EVIDENCE"
)

expected_6e2_classification = (
    "BOOTSTRAP_GENERATOR_NOT_RECOVERABLE_FROM_COMMITTED_PROVENANCE"
)


print(
    "6E1:"
)

print(
    " ",
    audit_6e1[
        "classification"
    ]
)

print(
    "\n6E2:"
)

print(
    " ",
    audit_6e2[
        "classification"
    ]
)


if audit_6e1[
    "classification"
] != expected_6e1_classification:

    raise RuntimeError(
        "Unexpected Stage26-6E1 provenance classification."
    )


if audit_6e2[
    "classification"
] != expected_6e2_classification:

    raise RuntimeError(
        "Unexpected Stage26-6E2 provenance classification."
    )


classification_evidence = audit_6e2[
    "classification_evidence"
]


if classification_evidence[
    "generator_found"
] is not False:

    raise RuntimeError(
        "Stage26-6E2 unexpectedly found a generator."
    )


if classification_evidence[
    "literal_seed_26042_certified"
] is not False:

    raise RuntimeError(
        "Historical literal seed was unexpectedly certified."
    )


if classification_evidence[
    "derived_bootstrap_seed_certified"
] is not False:

    raise RuntimeError(
        "Historical derived bootstrap seed was unexpectedly certified."
    )


print(
    "\nGenerator recoverable            : NO"
)

print(
    "Historical seed certified        : NO"
)

print(
    "Historical CIs protocol-certified: NO"
)


# =============================================================================
# 5. FROZEN ORIGINAL STATISTICAL PROTOCOL GATE
# =============================================================================

banner(
    "STAGE26-6F0 :: ORIGINAL FROZEN STATISTICAL PROTOCOL"
)


protocol = json.loads(
    PROTOCOL.read_text(
        encoding="utf-8"
    )
)


statistics = protocol[
    "statistics_protocol"
]

bootstrap = statistics[
    "bootstrap"
]

bootstrap_targets = statistics[
    "bootstrap_targets"
]


expected_bootstrap = {
    "enabled":
        True,

    "interval":
        "PERCENTILE_95_PERCENT",

    "replicates":
        2000,

    "rng":
        "numpy.random.default_rng_PCG64",

    "seed":
        26042,
}


if bootstrap != expected_bootstrap:

    raise RuntimeError(
        "Original frozen bootstrap configuration changed."
    )


expected_targets = [
    "p50_latency",
    "p95_latency",
    "p99_latency_when_n_gte_100",
    "median_throughput",
]


if bootstrap_targets != expected_targets:

    raise RuntimeError(
        "Original frozen bootstrap target list changed."
    )


print(
    json.dumps(
        bootstrap,
        indent=2,
        sort_keys=True,
    )
)


print(
    "\nTargets:"
)

for target in bootstrap_targets:

    print(
        " ",
        target
    )


print(
    "\nPR-AUC bootstrap target:",
    False
)


# =============================================================================
# 6. STAGE26-6D IMMUTABILITY GATE
# =============================================================================

banner(
    "STAGE26-6F0 :: PARETO IMMUTABILITY GATE"
)


frontier_6d = json.loads(
    FRONTIER_6D.read_text(
        encoding="utf-8"
    )
)

receipt_6d = json.loads(
    RECEIPT_6D.read_text(
        encoding="utf-8"
    )
)


if receipt_6d[
    "scientific_state"
][
    "point_estimate_frontiers_computed"
] is not True:

    raise RuntimeError(
        "Stage26-6D point-estimate frontier is not complete."
    )


if receipt_6d[
    "bootstrap_uncertainty_computed"
] is not False:

    raise RuntimeError(
        "Stage26-6D unexpectedly incorporated bootstrap uncertainty."
    )


if receipt_6d[
    "scientific_state"
][
    "cross_group_frontier_computed"
] is not False:

    raise RuntimeError(
        "Unexpected cross-group frontier."
    )


print(
    "Point-estimate frontier frozen : YES"
)

print(
    "Bootstrap used for membership  : NO"
)

print(
    "Correction may change membership: NO"
)

print(
    "Cross-group frontier           : NO"
)


# =============================================================================
# 7. CREATE CHECKPOINT DIRECTORY
# =============================================================================

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

PROVENANCE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# =============================================================================
# 8. DURABLY COPY 6E1 / 6E2 PROVENANCE
# =============================================================================

banner(
    "STAGE26-6F0 :: DURABILIZE PROVENANCE AUDITS"
)


shutil.copyfile(
    SOURCE_6E1,
    DURABLE_6E1,
)

shutil.copyfile(
    SOURCE_6E2,
    DURABLE_6E2,
)


if sha256_file(
    DURABLE_6E1
) != EXPECTED_6E1_SHA256:

    raise RuntimeError(
        "Durable 6E1 copy mismatch."
    )


if sha256_file(
    DURABLE_6E2
) != EXPECTED_6E2_SHA256:

    raise RuntimeError(
        "Durable 6E2 copy mismatch."
    )


print(
    "6E1 copy:",
    EXPECTED_6E1_SHA256
)

print(
    "6E2 copy:",
    EXPECTED_6E2_SHA256
)


# =============================================================================
# 9. FREEZE CORRECTION PROTOCOL
# =============================================================================

banner(
    "STAGE26-6F0 :: WRITE CORRECTION PROTOCOL"
)


correction_protocol = {
    "schema":
        "stage26_6f0_bootstrap_correction_protocol_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-6F0",

    "status":
        "FROZEN_BEFORE_CORRECTED_CI_COMPUTATION",

    "scientific_parent":
        EXPECTED_PARENT,

    "reason": {
        "historical_stage26_2_CIs_exist":
            True,

        "historical_stage26_2_CIs_protocol_certified":
            False,

        "historical_bootstrap_generator_recoverable":
            False,

        "historical_bootstrap_seed_certified":
            False,

        "final_provenance_classification":
            expected_6e2_classification,

        "correction_type":
            "DERIVED_SUMMARY_ONLY_STATISTICAL_CORRECTION",
    },

    "immutable_inputs": {
        "warm_raw": {
            "path":
                str(
                    WARM_RAW.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_WARM_RAW_SHA256,
        },

        "historical_summary": {
            "path":
                str(
                    WARM_SUMMARY.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_WARM_SUMMARY_SHA256,

            "policy":
                (
                    "Historical artifact preserved unchanged. "
                    "Existing CI fields are not protocol-certified."
                ),
        },

        "condition_status": {
            "path":
                str(
                    WARM_STATUS.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_WARM_STATUS_SHA256,
        },

        "stage26_measurement_protocol": {
            "path":
                str(
                    PROTOCOL.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_PROTOCOL_SHA256,
        },
    },

    "scope": {
        "conditions":
            "ALL_STAGE26_2_PASS_CONDITIONS",

        "failed_conditions":
            (
                "OOM and timeout conditions retain their frozen status; "
                "no bootstrap values are fabricated."
            ),

        "raw_measurement_reexecution":
            False,

        "model_inference_reexecution":
            False,

        "point_estimate_replacement":
            False,

        "historical_artifact_overwrite":
            False,
    },

    "bootstrap": {
        "replicates":
            2000,

        "interval":
            "PERCENTILE_95_PERCENT",

        "interval_percentiles": [
            2.5,
            97.5,
        ],

        "rng":
            "numpy.random.default_rng_PCG64",

        "literal_seed":
            26042,

        "derived_seed_allowed":
            False,

        "condition_local_rng_initialization":
            (
                "For every PASS condition independently execute "
                "np.random.default_rng(26042)."
            ),

        "condition_order_dependency":
            False,

        "metric_order_dependency":
            False,

        "sampling":
            "WITH_REPLACEMENT",

        "sample_size_per_replicate":
            "N_ORIGINAL_TIMED_OBSERVATIONS_FOR_CONDITION",

        "index_generation":
            (
                "rng.integers(0, n, size=(2000, n), endpoint=False)"
            ),

        "within_condition_pairing":
            (
                "The same 2000 x n resample-index matrix is reused for all "
                "frozen bootstrap targets within that condition."
            ),
    },

    "statistic_semantics": {
        "latency_source":
            "elapsed_ns",

        "latency_conversion":
            (
                "elapsed_ms = np.asarray(elapsed_ns, dtype=np.float64) "
                "/ 1_000_000.0"
            ),

        "conversion_before_statistic":
            True,

        "numpy_percentile_method":
            "linear",

        "p50_latency":
            (
                "For each bootstrap replicate compute np.percentile("
                "resampled_elapsed_ms, 50, method='linear')."
            ),

        "p95_latency":
            (
                "For each bootstrap replicate compute np.percentile("
                "resampled_elapsed_ms, 95, method='linear')."
            ),

        "p99_latency":
            (
                "Compute only when original n >= 100 using percentile 99; "
                "otherwise CI remains null."
            ),

        "median_throughput":
            (
                "Bootstrap preserved raw flows_per_second observations "
                "directly and compute np.median for each replicate."
            ),

        "maximum_latency":
            "DESCRIPTIVE_ONLY_NOT_BOOTSTRAPPED",

        "CI_construction":
            (
                "Apply np.percentile(bootstrap_statistic_vector, "
                "[2.5, 97.5], method='linear')."
            ),
    },

    "output_policy": {
        "new_output_directory":
            (
                "results/stage26_deployment_profiling/"
                "stage26_6f1_bootstrap_corrected_uncertainty"
            ),

        "historical_stage26_2_summary_modified":
            False,

        "historical_stage26_2_CIs_label":
            "HISTORICAL_NOT_PROTOCOL_CERTIFIED",

        "corrected_CIs_label":
            "PROTOCOL_CERTIFIED_DERIVED_UNCERTAINTY",

        "point_estimates_source":
            "ORIGINAL_IMMUTABLE_STAGE26_2_SUMMARY",

        "raw_measurements_source":
            "ORIGINAL_IMMUTABLE_STAGE26_2_WARM_RAW",
    },

    "pareto_policy": {
        "stage26_6d_frontier_sha256":
            EXPECTED_FRONTIER_6D_SHA256,

        "point_estimate_frontier_immutable":
            True,

        "corrected_uncertainty_may_change_frontier_membership":
            False,

        "cross_group_frontier_allowed":
            False,
    },

    "predictive_uncertainty_policy": {
        "PR_AUC_is_stage26_bootstrap_target":
            False,

        "PR_AUC_bootstrap_allowed":
            False,

        "predictive_metric_recomputation_allowed":
            False,

        "holdout_reopening_allowed":
            False,
    },

    "execution_guard": {
        "corrected_CI_computation_allowed_before_this_lock_is_git_anchored":
            False,

        "worker_source_must_match_locked_sha256":
            True,

        "input_hashes_must_match":
            True,

        "GPU_allowed":
            False,
    },
}


atomic_json(
    CORRECTION_PROTOCOL,
    correction_protocol,
)


# =============================================================================
# 10. FREEZE EXACT CORRECTION WORKER SOURCE — DO NOT EXECUTE
# =============================================================================

banner(
    "STAGE26-6F0 :: WRITE LOCKED CORRECTION WORKER"
)


worker_source = r'''
#!/usr/bin/env python3
"""
Stage26-6F1 locked derived-summary bootstrap correction worker.

IMPORTANT:
- This file is frozen in Stage26-6F0 before any corrected CI is computed.
- It consumes only immutable Stage26-2 raw timing observations.
- It performs no model loading, inference, timing, or predictive evaluation.
- It never overwrites historical Stage26-2 artifacts.
"""

from __future__ import annotations

import argparse
import csv
import json
from pathlib import Path

import numpy as np


BOOTSTRAP_REPLICATES = 2000
BOOTSTRAP_SEED = 26042
CI_PERCENTILES = (2.5, 97.5)
PERCENTILE_METHOD = "linear"


def load_csv(path: Path):

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def percentile_ci(values):

    values = np.asarray(
        values,
        dtype=np.float64,
    )

    result = np.percentile(
        values,
        CI_PERCENTILES,
        method=PERCENTILE_METHOD,
    )

    return [
        float(
            result[0]
        ),
        float(
            result[1]
        ),
    ]


def exact_point_estimates(
    elapsed_ms,
    throughput,
):

    n = int(
        elapsed_ms.size
    )

    result = {
        "p50_batch_latency_ms":
            float(
                np.percentile(
                    elapsed_ms,
                    50,
                    method=PERCENTILE_METHOD,
                )
            ),

        "p95_batch_latency_ms":
            float(
                np.percentile(
                    elapsed_ms,
                    95,
                    method=PERCENTILE_METHOD,
                )
            ),

        "p99_batch_latency_ms_if_n_gte_100":
            (
                float(
                    np.percentile(
                        elapsed_ms,
                        99,
                        method=PERCENTILE_METHOD,
                    )
                )
                if n >= 100
                else None
            ),

        "median_throughput_flows_per_second":
            float(
                np.median(
                    throughput
                )
            ),
    }

    return result


def corrected_bootstrap_for_condition(
    elapsed_ns,
    flows_per_second,
):

    elapsed_ns = np.asarray(
        elapsed_ns,
        dtype=np.float64,
    )

    throughput = np.asarray(
        flows_per_second,
        dtype=np.float64,
    )


    if elapsed_ns.ndim != 1:

        raise RuntimeError(
            "elapsed_ns must be one-dimensional."
        )


    if throughput.ndim != 1:

        raise RuntimeError(
            "flows_per_second must be one-dimensional."
        )


    if elapsed_ns.size != throughput.size:

        raise RuntimeError(
            "Latency and throughput observation counts differ."
        )


    n = int(
        elapsed_ns.size
    )


    if n <= 0:

        raise RuntimeError(
            "Cannot bootstrap an empty condition."
        )


    # Historical point-estimate semantics established by Stage26-6E:
    # convert raw nanoseconds to float64 milliseconds BEFORE percentile.
    elapsed_ms = (
        elapsed_ns
        /
        1_000_000.0
    )


    # FROZEN CORRECTION:
    # literal condition-local seed; absolutely no derived seed.
    rng = np.random.default_rng(
        BOOTSTRAP_SEED
    )


    indices = rng.integers(
        0,
        n,
        size=(
            BOOTSTRAP_REPLICATES,
            n,
        ),
        endpoint=False,
        dtype=np.int64,
    )


    # Same resample matrix reused for all frozen targets.
    boot_latency = elapsed_ms[
        indices
    ]

    boot_throughput = throughput[
        indices
    ]


    p50_rep = np.percentile(
        boot_latency,
        50,
        axis=1,
        method=PERCENTILE_METHOD,
    )

    p95_rep = np.percentile(
        boot_latency,
        95,
        axis=1,
        method=PERCENTILE_METHOD,
    )


    if n >= 100:

        p99_rep = np.percentile(
            boot_latency,
            99,
            axis=1,
            method=PERCENTILE_METHOD,
        )

    else:

        p99_rep = None


    median_throughput_rep = np.median(
        boot_throughput,
        axis=1,
    )


    point = exact_point_estimates(
        elapsed_ms,
        throughput,
    )


    result = {
        "n":
            n,

        "bootstrap_replicates":
            BOOTSTRAP_REPLICATES,

        "bootstrap_seed":
            BOOTSTRAP_SEED,

        "rng":
            "numpy.random.default_rng_PCG64",

        "sampling":
            "WITH_REPLACEMENT",

        "same_resample_indices_for_all_targets":
            True,

        "point_estimate_integrity":
            point,

        "corrected_ci95": {
            "p50_batch_latency_ms":
                percentile_ci(
                    p50_rep
                ),

            "p95_batch_latency_ms":
                percentile_ci(
                    p95_rep
                ),

            "p99_batch_latency_ms_if_n_gte_100":
                (
                    percentile_ci(
                        p99_rep
                    )
                    if p99_rep is not None
                    else None
                ),

            "median_throughput_flows_per_second":
                percentile_ci(
                    median_throughput_rep
                ),
        },
    }


    return result


def main():

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--raw-csv",
        required=True,
    )

    parser.add_argument(
        "--summary-csv",
        required=True,
    )

    parser.add_argument(
        "--condition-status-csv",
        required=True,
    )

    parser.add_argument(
        "--output-json",
        required=True,
    )

    args = parser.parse_args()


    raw_path = Path(
        args.raw_csv
    )

    summary_path = Path(
        args.summary_csv
    )

    status_path = Path(
        args.condition_status_csv
    )

    output_path = Path(
        args.output_json
    )


    raw_rows = load_csv(
        raw_path
    )

    summary_rows = load_csv(
        summary_path
    )

    status_rows = load_csv(
        status_path
    )


    summary_by_condition = {
        row[
            "condition_id"
        ]:
            row
        for row in summary_rows
    }


    status_by_condition = {
        row[
            "condition_id"
        ]:
            row
        for row in status_rows
    }


    condition_ids = sorted(
        status_by_condition,
        key=lambda value: int(
            value.split(
                "_"
            )[
                -1
            ]
        ),
    )


    output_conditions = []


    for condition_id in condition_ids:

        status_row = status_by_condition[
            condition_id
        ]

        status = status_row[
            "status"
        ]


        if status != "PASS":

            output_conditions.append(
                {
                    "condition_id":
                        condition_id,

                    "status":
                        status,

                    "bootstrap_computed":
                        False,

                    "reason":
                        "NON_PASS_FROZEN_RESOURCE_OUTCOME",
                }
            )

            continue


        raw_condition = [
            row
            for row in raw_rows
            if row[
                "condition_id"
            ]
            ==
            condition_id
        ]


        if not raw_condition:

            raise RuntimeError(
                f"{condition_id}: PASS but raw observations absent."
            )


        if any(
            row[
                "status"
            ]
            !=
            "PASS"
            for row in raw_condition
        ):

            raise RuntimeError(
                f"{condition_id}: non-PASS raw row found."
            )


        summary_row = summary_by_condition[
            condition_id
        ]


        expected_n = int(
            summary_row[
                "n"
            ]
        )


        if len(
            raw_condition
        ) != expected_n:

            raise RuntimeError(
                f"{condition_id}: raw n mismatch."
            )


        elapsed_ns = [
            int(
                row[
                    "elapsed_ns"
                ]
            )
            for row in raw_condition
        ]

        throughput = [
            float(
                row[
                    "flows_per_second"
                ]
            )
            for row in raw_condition
        ]


        corrected = corrected_bootstrap_for_condition(
            elapsed_ns,
            throughput,
        )


        # -------------------------------------------------------------
        # Integrity gate:
        # corrected procedure must reproduce immutable point estimates.
        # It is forbidden to replace point estimates.
        # -------------------------------------------------------------

        point = corrected[
            "point_estimate_integrity"
        ]


        stored_p50 = float(
            summary_row[
                "p50_batch_latency_ms"
            ]
        )

        stored_p95 = float(
            summary_row[
                "p95_batch_latency_ms"
            ]
        )

        stored_tp = float(
            summary_row[
                "median_throughput_flows_per_second"
            ]
        )


        if point[
            "p50_batch_latency_ms"
        ] != stored_p50:

            raise RuntimeError(
                f"{condition_id}: p50 point-estimate integrity failed."
            )


        if point[
            "p95_batch_latency_ms"
        ] != stored_p95:

            raise RuntimeError(
                f"{condition_id}: p95 point-estimate integrity failed."
            )


        if point[
            "median_throughput_flows_per_second"
        ] != stored_tp:

            raise RuntimeError(
                f"{condition_id}: throughput point-estimate integrity failed."
            )


        if expected_n >= 100:

            stored_p99 = float(
                summary_row[
                    "p99_batch_latency_ms_if_n_gte_100"
                ]
            )


            if point[
                "p99_batch_latency_ms_if_n_gte_100"
            ] != stored_p99:

                raise RuntimeError(
                    f"{condition_id}: p99 point-estimate integrity failed."
                )


        else:

            if (
                summary_row[
                    "p99_batch_latency_ms_if_n_gte_100"
                ]
                not in {
                    "",
                    None,
                }
            ):

                raise RuntimeError(
                    f"{condition_id}: unexpected historical p99 for n<100."
                )


        output_conditions.append(
            {
                "condition_id":
                    condition_id,

                "target_id":
                    summary_row[
                        "target_id"
                    ],

                "comparison_group":
                    summary_row[
                        "comparison_group"
                    ],

                "hardware_mode":
                    summary_row[
                        "hardware_mode"
                    ],

                "thread_count":
                    int(
                        summary_row[
                            "thread_count"
                        ]
                    ),

                "batch_size":
                    int(
                        summary_row[
                            "batch_size"
                        ]
                    ),

                "status":
                    "PASS",

                "bootstrap_computed":
                    True,

                "historical_CI_status":
                    "HISTORICAL_NOT_PROTOCOL_CERTIFIED",

                "corrected_CI_status":
                    "PROTOCOL_CERTIFIED_DERIVED_UNCERTAINTY",

                "historical_ci95": {
                    "p50_batch_latency_ms": [
                        float(
                            summary_row[
                                "p50_ci95_low_ms"
                            ]
                        ),
                        float(
                            summary_row[
                                "p50_ci95_high_ms"
                            ]
                        ),
                    ],

                    "p95_batch_latency_ms": [
                        float(
                            summary_row[
                                "p95_ci95_low_ms"
                            ]
                        ),
                        float(
                            summary_row[
                                "p95_ci95_high_ms"
                            ]
                        ),
                    ],

                    "p99_batch_latency_ms_if_n_gte_100":
                        (
                            [
                                float(
                                    summary_row[
                                        "p99_ci95_low_ms_if_n_gte_100"
                                    ]
                                ),
                                float(
                                    summary_row[
                                        "p99_ci95_high_ms_if_n_gte_100"
                                    ]
                                ),
                            ]
                            if expected_n >= 100
                            else None
                        ),

                    "median_throughput_flows_per_second": [
                        float(
                            summary_row[
                                "median_throughput_ci95_low"
                            ]
                        ),
                        float(
                            summary_row[
                                "median_throughput_ci95_high"
                            ]
                        ),
                    ],
                },

                **corrected,
            }
        )


    payload = {
        "schema":
            "stage26_6f1_bootstrap_corrected_uncertainty_v1",

        "bootstrap_protocol": {
            "replicates":
                BOOTSTRAP_REPLICATES,

            "seed":
                BOOTSTRAP_SEED,

            "rng":
                "numpy.random.default_rng_PCG64",

            "condition_local_seed_reset":
                True,

            "same_resample_indices_for_all_targets":
                True,

            "percentile_method":
                PERCENTILE_METHOD,

            "ci_percentiles":
                list(
                    CI_PERCENTILES
                ),
        },

        "conditions":
            output_conditions,
    }


    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    with output_path.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )


if __name__ == "__main__":
    main()
'''


worker_source = textwrap.dedent(
    worker_source
).lstrip()


atomic_text(
    CORRECTION_WORKER,
    worker_source,
)


worker_sha = sha256_file(
    CORRECTION_WORKER
)


print(
    "Locked worker SHA256:"
)

print(
    " ",
    worker_sha
)


# =============================================================================
# 11. WRITE STATISTICAL ERRATUM RECORD
# =============================================================================

banner(
    "STAGE26-6F0 :: WRITE STATISTICAL ERRATUM RECORD"
)


erratum_record = {
    "schema":
        "stage26_6f0_statistical_erratum_record_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-6F0",

    "classification":
        "DERIVED_SUMMARY_IMPLEMENTATION_PROVENANCE_ERRATUM",

    "affected_artifacts": {
        "historical_stage26_2_bootstrap_CI_fields":
            True,

        "stage26_2_raw_timing_measurements":
            False,

        "stage26_2_point_estimates":
            False,

        "stage26_2_condition_outcomes":
            False,

        "stage26_6d_pareto_membership":
            False,

        "predictive_metrics":
            False,
    },

    "finding": (
        "The Stage26-2 bootstrap confidence intervals were durably stored, "
        "but the exact CI-generating implementation and bootstrap-specific "
        "seed provenance were not recoverable from committed repository "
        "history. The preserved Stage26-2 worker RNG calls are exclusively "
        "associated with deterministic synthetic model-input generation."
    ),

    "publication_policy": {
        "historical_CIs":
            "RETAIN_FOR_AUDIT_NOT_PROTOCOL_CERTIFIED",

        "corrected_CIs":
            (
                "USE_FOR_FINAL_STAGE26_TIMING_UNCERTAINTY_AFTER_EXECUTION_OF_"
                "THE_GIT_ANCHORED_6F0_WORKER"
            ),

        "raw_measurements":
            "UNCHANGED",

        "point_estimates":
            "UNCHANGED",

        "pareto_frontier":
            "UNCHANGED_POINT_ESTIMATE_FRONTIER",

        "PR_AUC_bootstrap":
            "NOT_AUTHORIZED",
    },

    "provenance": {
        "stage26_6e1_sha256":
            EXPECTED_6E1_SHA256,

        "stage26_6e2_sha256":
            EXPECTED_6E2_SHA256,

        "final_classification":
            expected_6e2_classification,
    },

    "correction_protocol_path":
        str(
            CORRECTION_PROTOCOL.relative_to(
                REPO
            )
        ),

    "correction_worker_path":
        str(
            CORRECTION_WORKER.relative_to(
                REPO
            )
        ),

    "correction_worker_sha256":
        worker_sha,

    "corrected_values_seen_before_freeze":
        False,

    "bootstrap_executed_during_freeze":
        False,

    "new_measurement_performed":
        False,

    "GPU_used":
        False,
}


atomic_json(
    ERRATUM_RECORD,
    erratum_record,
)


# =============================================================================
# 12. FREEZE RECEIPT
# =============================================================================

protocol_sha = sha256_file(
    CORRECTION_PROTOCOL
)

erratum_sha = sha256_file(
    ERRATUM_RECORD
)


freeze_receipt = {
    "schema":
        "stage26_6f0_bootstrap_correction_freeze_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-6F0",

    "status":
        "PASS_BOOTSTRAP_CORRECTION_PROTOCOL_FROZEN_BEFORE_EXECUTION",

    "scientific_parent":
        EXPECTED_PARENT,

    "source_identity": {
        "measurement_protocol_sha256":
            EXPECTED_PROTOCOL_SHA256,

        "stage26_2_warm_raw_sha256":
            EXPECTED_WARM_RAW_SHA256,

        "stage26_2_historical_summary_sha256":
            EXPECTED_WARM_SUMMARY_SHA256,

        "stage26_2_condition_status_sha256":
            EXPECTED_WARM_STATUS_SHA256,

        "stage26_6d_frontier_sha256":
            EXPECTED_FRONTIER_6D_SHA256,

        "stage26_6e1_sha256":
            EXPECTED_6E1_SHA256,

        "stage26_6e2_sha256":
            EXPECTED_6E2_SHA256,
    },

    "locked_artifacts": {
        "correction_protocol_sha256":
            protocol_sha,

        "correction_worker_sha256":
            worker_sha,

        "erratum_record_sha256":
            erratum_sha,
    },

    "frozen_bootstrap": {
        "replicates":
            2000,

        "seed":
            26042,

        "rng":
            "numpy.random.default_rng_PCG64",

        "derived_seeds":
            False,

        "condition_local_literal_seed_reset":
            True,

        "same_resample_indices_for_all_condition_statistics":
            True,

        "latency_statistic_unit":
            "MILLISECONDS_BEFORE_PERCENTILE",

        "percentile_method":
            "linear",

        "interval":
            "PERCENTILE_95_PERCENT",
    },

    "scope":
        "ALL_STAGE26_2_PASS_CONDITIONS",

    "scientific_state": {
        "corrected_CI_computed":
            False,

        "bootstrap_executed":
            False,

        "historical_CI_overwritten":
            False,

        "raw_measurement_modified":
            False,

        "point_estimate_modified":
            False,

        "predictive_metric_recomputed":
            False,

        "PR_AUC_bootstrap_computed":
            False,

        "holdout_reopened":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "Pareto_frontier_recomputed":
            False,

        "Pareto_frontier_changed":
            False,

        "cross_group_frontier_computed":
            False,

        "PCAP_accessed":
            False,

        "Release_corpus_accessed":
            False,

        "GPU_used":
            False,
    },

    "next_permitted_action":
        (
            "Only after this checkpoint is remotely Git-verified, execute the "
            "locked correction worker against the immutable Stage26-2 raw "
            "timing observations and persist corrected derived uncertainty "
            "outputs in a new Stage26-6F1 checkpoint."
        ),
}


atomic_json(
    FREEZE_RECEIPT,
    freeze_receipt,
)


receipt_sha = sha256_file(
    FREEZE_RECEIPT
)


# =============================================================================
# 13. MANIFEST
# =============================================================================

package_files = [
    CORRECTION_PROTOCOL,
    CORRECTION_WORKER,
    ERRATUM_RECORD,
    FREEZE_RECEIPT,
    DURABLE_6E1,
    DURABLE_6E2,
]


manifest_rows = []


for path in package_files:

    manifest_rows.append(
        {
            "repo_relative_path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest = {
    "schema":
        "stage26_6f0_bootstrap_correction_lock_manifest_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-6F0",

    "status":
        "READY_FOR_GIT_ANCHOR",

    "scientific_parent":
        EXPECTED_PARENT,

    "commit_subject":
        COMMIT_SUBJECT,

    "correction_protocol_sha256":
        protocol_sha,

    "correction_worker_sha256":
        worker_sha,

    "erratum_record_sha256":
        erratum_sha,

    "freeze_receipt_sha256":
        receipt_sha,

    "file_count_excluding_manifest":
        len(
            manifest_rows
        ),

    "files":
        manifest_rows,

    "scientific_state": {
        "protocol_frozen":
            True,

        "worker_frozen":
            True,

        "corrected_values_seen":
            False,

        "bootstrap_executed":
            False,

        "new_measurement_performed":
            False,

        "Pareto_changed":
            False,

        "GPU_used":
            False,
    },
}


atomic_json(
    MANIFEST,
    manifest,
)


manifest_sha = sha256_file(
    MANIFEST
)


print(
    "Correction protocol:",
    protocol_sha
)

print(
    "Correction worker  :",
    worker_sha
)

print(
    "Erratum record     :",
    erratum_sha
)

print(
    "Freeze receipt     :",
    receipt_sha
)

print(
    "Manifest           :",
    manifest_sha
)


# =============================================================================
# 14. LOCAL PACKAGE AUDIT
# =============================================================================

banner(
    "STAGE26-6F0 :: LOCAL PACKAGE AUDIT"
)


for row in manifest_rows:

    path = (
        REPO
        /
        row[
            "repo_relative_path"
        ]
    )

    size = int(
        path.stat().st_size
    )

    digest = sha256_file(
        path
    )

    passed = (
        size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        digest
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{size:10,d} B "
        f"{digest} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Local Stage26-6F0 package audit failed."
        )


# =============================================================================
# 15. STATIC WORKER SAFETY AUDIT
# =============================================================================

banner(
    "STAGE26-6F0 :: STATIC WORKER SAFETY AUDIT"
)


locked_worker_text = CORRECTION_WORKER.read_text(
    encoding="utf-8"
)


required_worker_fragments = [
    "BOOTSTRAP_REPLICATES = 2000",
    "BOOTSTRAP_SEED = 26042",
    "np.random.default_rng(",
    "BOOTSTRAP_SEED",
    "endpoint=False",
    "method=PERCENTILE_METHOD",
    "elapsed_ns",
    "1_000_000.0",
    "flows_per_second",
    "same_resample_indices_for_all_targets",
]


for fragment in required_worker_fragments:

    passed = (
        fragment
        in
        locked_worker_text
    )


    print(
        f"{fragment:52s} "
        f"{'PASS' if passed else 'FAIL'}"
    )


    if not passed:

        raise RuntimeError(
            f"Locked worker missing required fragment: {fragment}"
        )


forbidden_fragments = [
    "torch.load",
    "joblib.load",
    "xgboost",
    "lightgbm",
    "catboost",
    "average_precision_score",
    "roc_auc_score",
    "predict_proba",
    ".predict(",
    "pcap",
    "cuda",
]


for fragment in forbidden_fragments:

    found = (
        fragment.lower()
        in
        locked_worker_text.lower()
    )


    print(
        f"FORBID {fragment:45s} "
        f"{'FAIL' if found else 'PASS'}"
    )


    if found:

        raise RuntimeError(
            f"Forbidden capability found in correction worker: {fragment}"
        )


# =============================================================================
# 16. GIT CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-6F0 :: GIT CHANGE AUDIT"
)


repo_status = git(
    "status",
    "--porcelain",
)


print(
    repo_status
)


if not repo_status:

    raise RuntimeError(
        "Expected uncommitted Stage26-6F0 package."
    )


unexpected = []


for line in repo_status.splitlines():

    relpath = line[
        3:
    ]


    if not relpath.startswith(
        str(
            CHECKPOINT_REL
        )
        +
        "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository changes:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 17. GIT IDENTITY
# =============================================================================

author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_PARENT,
)

author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_PARENT,
)


if (
    not author_name.strip()
    or
    "@"
    not in
    author_email
):

    raise RuntimeError(
        "Could not recover valid Git identity."
    )


git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


print(
    "\nGit author:"
)

print(
    " ",
    author_name,
    "<" + author_email + ">"
)


# =============================================================================
# 18. COMMIT
# =============================================================================

banner(
    "STAGE26-6F0 :: COMMIT"
)


git(
    "add",
    str(
        CHECKPOINT_REL
    ),
)


print(
    git(
        "diff",
        "--cached",
        "--name-status",
    )
)


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-6F0 commit parent mismatch."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Stage26-6F0 commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository not clean after Stage26-6F0 commit."
    )


# =============================================================================
# 19. PUSH
# =============================================================================

banner(
    "STAGE26-6F0 :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    push_result = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
        text=True,
    )


    print(
        push_result.stdout.strip()
    )


github_token = None


# =============================================================================
# 20. REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-6F0 :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "origin/main did not advance to Stage26-6F0."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote Stage26-6F0 parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote Stage26-6F0 subject mismatch."
    )


# =============================================================================
# 21. REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-6F0 :: REMOTE BYTE VERIFICATION"
)


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    str(
        MANIFEST.relative_to(
            REPO
        )
    ),
)


remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "Remote manifest SHA256:"
)

print(
    " ",
    remote_manifest_sha
)


if remote_manifest_sha != manifest_sha:

    raise RuntimeError(
        "Remote Stage26-6F0 manifest mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


for row in remote_manifest[
    "files"
]:

    data = git_blob_bytes(
        "origin/main",
        row[
            "repo_relative_path"
        ],
    )

    actual_size = len(
        data
    )

    actual_sha = sha256_bytes(
        data
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Remote Stage26-6F0 byte verification failed."
        )


# =============================================================================
# 22. REMOTE SCIENTIFIC VERIFICATION
# =============================================================================

banner(
    "STAGE26-6F0 :: REMOTE SCIENTIFIC VERIFICATION"
)


remote_protocol = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            CORRECTION_PROTOCOL.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_receipt = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            FREEZE_RECEIPT.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_erratum = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            ERRATUM_RECORD.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_worker_bytes = git_blob_bytes(
    "origin/main",
    str(
        CORRECTION_WORKER.relative_to(
            REPO
        )
    ),
)


remote_checks = {
    "frozen before CI computation":
        (
            remote_protocol[
                "status"
            ]
            ==
            "FROZEN_BEFORE_CORRECTED_CI_COMPUTATION"
        ),

    "2,000 replicates":
        (
            remote_protocol[
                "bootstrap"
            ][
                "replicates"
            ]
            ==
            2000
        ),

    "literal seed 26042":
        (
            remote_protocol[
                "bootstrap"
            ][
                "literal_seed"
            ]
            ==
            26042
        ),

    "derived seeds forbidden":
        (
            remote_protocol[
                "bootstrap"
            ][
                "derived_seed_allowed"
            ]
            is False
        ),

    "condition-local reset":
        (
            "np.random.default_rng(26042)"
            in
            remote_protocol[
                "bootstrap"
            ][
                "condition_local_rng_initialization"
            ]
        ),

    "same resample indices":
        (
            remote_receipt[
                "frozen_bootstrap"
            ][
                "same_resample_indices_for_all_condition_statistics"
            ]
            is True
        ),

    "milliseconds before percentile":
        (
            remote_receipt[
                "frozen_bootstrap"
            ][
                "latency_statistic_unit"
            ]
            ==
            "MILLISECONDS_BEFORE_PERCENTILE"
        ),

    "worker SHA":
        (
            sha256_bytes(
                remote_worker_bytes
            )
            ==
            worker_sha
        ),

    "historical CI overwrite false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "historical_CI_overwritten"
            ]
            is False
        ),

    "bootstrap not executed":
        (
            remote_receipt[
                "scientific_state"
            ][
                "bootstrap_executed"
            ]
            is False
        ),

    "corrected CI not computed":
        (
            remote_receipt[
                "scientific_state"
            ][
                "corrected_CI_computed"
            ]
            is False
        ),

    "Pareto unchanged":
        (
            remote_receipt[
                "scientific_state"
            ][
                "Pareto_frontier_changed"
            ]
            is False
        ),

    "PR-AUC bootstrap false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "PR_AUC_bootstrap_computed"
            ]
            is False
        ),

    "GPU false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "GPU_used"
            ]
            is False
        ),

    "erratum affects CI only":
        (
            remote_erratum[
                "affected_artifacts"
            ][
                "historical_stage26_2_bootstrap_CI_fields"
            ]
            is True
            and
            remote_erratum[
                "affected_artifacts"
            ][
                "stage26_2_raw_timing_measurements"
            ]
            is False
            and
            remote_erratum[
                "affected_artifacts"
            ][
                "stage26_2_point_estimates"
            ]
            is False
        ),
}


for name, passed in remote_checks.items():

    print(
        f"{name:42s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    remote_checks.values()
):

    raise RuntimeError(
        "Remote Stage26-6F0 scientific verification failed."
    )


# =============================================================================
# 23. FINAL CLOSURE
# =============================================================================

banner(
    "STAGE26-6F0 BOOTSTRAP CORRECTION PROTOCOL FREEZE COMPLETE"
)


final_status = git(
    "status",
    "--porcelain",
)


print(
    "NEW DURABLE COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nPARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nFINAL PROVENANCE FINDING:"
)

print(
    " ",
    expected_6e2_classification
)


print(
    "\nLOCKED CORRECTION:"
)

print(
    "  scope                    : ALL Stage26-2 PASS conditions"
)

print(
    "  replicates               : 2,000"
)

print(
    "  RNG                      : numpy.random.default_rng / PCG64"
)

print(
    "  literal seed             : 26042"
)

print(
    "  derived seeds            : FORBIDDEN"
)

print(
    "  seed reset               : independently per condition"
)

print(
    "  same indices per metrics : YES"
)

print(
    "  latency conversion       : ns -> float64 ms BEFORE percentile"
)

print(
    "  CI                       : percentile [2.5,97.5], method=linear"
)

print(
    "  p99                      : only n >= 100"
)

print(
    "  median throughput        : bootstrap raw flows_per_second"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  correction protocol frozen       : YES"
)

print(
    "  correction worker frozen         : YES"
)

print(
    "  corrected values seen            : NO"
)

print(
    "  bootstrap executed               : NO"
)

print(
    "  CI calculated                    : NO"
)

print(
    "  historical Stage26-2 overwritten : NO"
)

print(
    "  raw measurements modified        : NO"
)

print(
    "  point estimates modified         : NO"
)

print(
    "  Pareto frontier modified         : NO"
)

print(
    "  PR-AUC bootstrap                 : NO"
)

print(
    "  holdout reopened                 : NO"
)

print(
    "  model loaded                     : NO"
)

print(
    "  inference                        : NO"
)

print(
    "  timing                           : NO"
)

print(
    "  GPU                              : NO"
)


print(
    "\nHASHES:"
)

print(
    "  correction protocol:",
    protocol_sha
)

print(
    "  correction worker  :",
    worker_sha
)

print(
    "  erratum record     :",
    erratum_sha
)

print(
    "  freeze receipt     :",
    receipt_sha
)

print(
    "  manifest           :",
    manifest_sha
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  commit             : PASS"
)

print(
    "  parent             : PASS"
)

print(
    "  package bytes      : PASS"
)

print(
    "  protocol           : PASS"
)

print(
    "  worker source      : PASS"
)

print(
    "  scientific content : PASS"
)


print(
    "\nRepo clean:",
    final_status == ""
)


if final_status:

    raise RuntimeError(
        "Repository not clean after Stage26-6F0."
    )


print(
    "\nNEXT:"
)

print(
    "  Stage26-6F1 may now execute ONLY the remotely anchored"
)

print(
    "  correction worker against immutable Stage26-2 raw timing"
)

print(
    "  observations and generate corrected derived uncertainty."
)

print(
    "  No model inference or timing rerun is permitted."
)

print(
    "  Stage26-6D Pareto membership remains immutable."
)

print(
    "  No GPU yet."
)


STAGE26-6F0 :: DURABLE SCIENTIFIC PARENT
Expected parent: ff9d329785c6cd30d273729356f402e59dc4e844
Local HEAD     : ff9d329785c6cd30d273729356f402e59dc4e844
origin/main    : ff9d329785c6cd30d273729356f402e59dc4e844
Repo clean     : True

STAGE26-6F0 :: IMMUTABLE SOURCE IDENTITY
measurement protocol               PASS d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
Stage26-2 warm raw                 PASS 78c58289ccfc4598966d6516201028f43c4b1d16d8bd0268a0cf58129d4179fa
Stage26-2 historical summary       PASS 75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232
Stage26-2 condition status         PASS df146563826992cb57702e589e4cf5d875ecfdbbf79533a731405e8eb738e7af
Stage26-2 receipt                  PASS b4b2623eabde7dd6b9acc250357a9f1ad61fa342c1dd496dcb634d4bfaca4d15
Stage26-6D frontier                PASS 367b3d34ddb4dd125dcbe3db71291b5576b4521640cd2f11d76e50dc247fc6b5
Stage26-6D receipt                 PASS d898f08dc45cdb625efa5c000d76c4414a245d2091dc922743

In [22]:
# =============================================================================
# STAGE26-6F1
# EXECUTE REMOTELY-ANCHORED BOOTSTRAP CI CORRECTION
# PERSIST + COMMIT + PUSH + REMOTE VERIFY
#
# DURABLE SCIENTIFIC PARENT:
#   94a2c5c290f67e3f9dda10a049e6a5318e912bdd
#
# LOCKED WORKER:
#   SHA256
#   9f4e6b6adb9a020128a3d935d7fc4e37412fa3b944d5d1ac7119c716a9cdc91b
#
# LOCKED PROTOCOL:
#   SHA256
#   9e267bf526304ce2c29ed3a74e8f2be29f0320de613860a3ea47752a9f9192f8
#
# PURPOSE
# -------
# Execute ONLY the Stage26-6F0 Git-anchored derived-summary correction worker
# against immutable Stage26-2 raw timing observations.
#
# CORRECTION:
#   - ALL Stage26-2 PASS conditions
#   - 2,000 bootstrap replicates
#   - np.random.default_rng(26042) / PCG64
#   - literal condition-local seed reset
#   - no derived seeds
#   - same bootstrap indices for all metrics within a condition
#   - elapsed_ns -> float64 milliseconds BEFORE latency statistic
#   - percentile method="linear"
#   - CI = [2.5, 97.5] percentiles of bootstrap statistic
#   - p99 only when n >= 100
#   - median throughput bootstrap from preserved flows_per_second
#
# THIS CELL ALSO:
#   - verifies original point estimates remain bit-for-bit unchanged;
#   - compares historical uncertified vs corrected certified CIs;
#   - extracts corrected p95 uncertainty for the six frozen Pareto points;
#   - DOES NOT recompute Pareto dominance or membership;
#   - preserves OOM / TIMEOUT conditions unchanged;
#   - commits / pushes / remotely verifies corrected derived results.
#
# NO:
#   - model loading
#   - model inference
#   - timing remeasurement
#   - PR-AUC bootstrap
#   - holdout access
#   - predictive metric recomputation
#   - Pareto recomputation
#   - PCAP
#   - Release corpus
#   - GPU
# =============================================================================

from __future__ import annotations

import csv
import json
import os
import sys
import stat
import hashlib
import subprocess
import tempfile
from collections import Counter
from pathlib import Path
from datetime import datetime, timezone

import numpy as np


# =============================================================================
# 0. CONSTANTS / FROZEN IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "94a2c5c290f67e3f9dda10a049e6a5318e912bdd"
)

COMMIT_SUBJECT = (
    "stage26: anchor corrected CPU timing uncertainty"
)


# -----------------------------------------------------------------------------
# Stage26-6F0 correction lock
# -----------------------------------------------------------------------------

LOCK_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_6f0_bootstrap_correction_protocol_lock"
)

CORRECTION_PROTOCOL = (
    LOCK_DIR
    / "stage26_6f0_bootstrap_correction_protocol.json"
)

CORRECTION_WORKER = (
    LOCK_DIR
    / "stage26_6f0_bootstrap_correction_worker.py"
)

ERRATUM_RECORD = (
    LOCK_DIR
    / "stage26_6f0_statistical_erratum_record.json"
)

FREEZE_RECEIPT = (
    LOCK_DIR
    / "stage26_6f0_bootstrap_correction_freeze_receipt.json"
)

LOCK_MANIFEST = (
    LOCK_DIR
    / "stage26_6f0_bootstrap_correction_lock_manifest.json"
)


EXPECTED_PROTOCOL_SHA256 = (
    "9e267bf526304ce2c29ed3a74e8f2be29f0320de613860a3ea47752a9f9192f8"
)

EXPECTED_WORKER_SHA256 = (
    "9f4e6b6adb9a020128a3d935d7fc4e37412fa3b944d5d1ac7119c716a9cdc91b"
)

EXPECTED_ERRATUM_SHA256 = (
    "59dc41df1f805180697a04a6be60c2e36d193d5623a02700713e3b5827a437f3"
)

EXPECTED_FREEZE_RECEIPT_SHA256 = (
    "bee8b8922497ba23f8ec73e46ccfd4c863baad869b4dbfca90f081ce701b77c2"
)

EXPECTED_LOCK_MANIFEST_SHA256 = (
    "bb1509d721c8e26b14514d27790b534011ffac3f2452c712859a5f2dd4921c01"
)


# -----------------------------------------------------------------------------
# Immutable Stage26-2 measurement artifacts
# -----------------------------------------------------------------------------

WARM_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_2_cpu_warm_inference"
)

WARM_RAW = (
    WARM_DIR
    / "stage26_2_warm_raw.csv"
)

WARM_SUMMARY = (
    WARM_DIR
    / "stage26_2_warm_summary.csv"
)

WARM_STATUS = (
    WARM_DIR
    / "stage26_2_condition_status.csv"
)


EXPECTED_WARM_RAW_SHA256 = (
    "78c58289ccfc4598966d6516201028f43c4b1d16d8bd0268a0cf58129d4179fa"
)

EXPECTED_WARM_SUMMARY_SHA256 = (
    "75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232"
)

EXPECTED_WARM_STATUS_SHA256 = (
    "df146563826992cb57702e589e4cf5d875ecfdbbf79533a731405e8eb738e7af"
)


# -----------------------------------------------------------------------------
# Immutable Stage26-6D Pareto result
# -----------------------------------------------------------------------------

DIR_6D = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_6d_cpu_pareto_point_estimate"
)

FRONTIER_6D = (
    DIR_6D
    / "stage26_6d_point_estimate_frontiers.json"
)

EXPECTED_FRONTIER_6D_SHA256 = (
    "367b3d34ddb4dd125dcbe3db71291b5576b4521640cd2f11d76e50dc247fc6b5"
)


# -----------------------------------------------------------------------------
# Stage26-6F1 output package
# -----------------------------------------------------------------------------

CHECKPOINT_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_6f1_bootstrap_corrected_uncertainty"
)

CHECKPOINT_DIR = (
    REPO
    / CHECKPOINT_REL
)

CORRECTED_JSON = (
    CHECKPOINT_DIR
    / "stage26_6f1_bootstrap_corrected_uncertainty.json"
)

CORRECTED_CSV = (
    CHECKPOINT_DIR
    / "stage26_6f1_corrected_timing_uncertainty.csv"
)

PARETO_CSV = (
    CHECKPOINT_DIR
    / "stage26_6f1_six_point_p95_uncertainty.csv"
)

EXECUTION_RECEIPT = (
    CHECKPOINT_DIR
    / "stage26_6f1_correction_execution_receipt.json"
)

MANIFEST = (
    CHECKPOINT_DIR
    / "stage26_6f1_corrected_uncertainty_manifest.json"
)


# =============================================================================
# 1. FROZEN SIX PARETO CONDITIONS
# =============================================================================

PARETO_CONDITIONS = {
    "STAGE16_XGBOOST_TUNED":
        "CPUCOND_001",

    "STAGE16_LIGHTGBM_TUNED":
        "CPUCOND_011",

    "STAGE16_CATBOOST_TUNED":
        "CPUCOND_021",

    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING":
        "CPUCOND_031",

    "STAGE20_MASKED_CNN_V1":
        "CPUCOND_041",

    "STAGE21_MASKED_VIT_V1":
        "CPUCOND_051",
}


# =============================================================================
# 2. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def load_csv(path):

    with Path(path).open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def atomic_json(
    path,
    payload,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        +
        ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        text=False,
    )

    return bytes(
        p.stdout
    )


# =============================================================================
# 3. DURABLE GIT GATE
# =============================================================================

banner(
    "STAGE26-6F1 :: DURABLE SCIENTIFIC PARENT"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-6F1 parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before corrected CI execution."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if CHECKPOINT_DIR.exists():

    raise RuntimeError(
        "Stage26-6F1 checkpoint already exists."
    )


# =============================================================================
# 4. EXACT LOCK / INPUT IDENTITY
# =============================================================================

banner(
    "STAGE26-6F1 :: LOCKED IMPLEMENTATION / INPUT IDENTITY"
)


checks = [
    (
        "correction protocol",
        CORRECTION_PROTOCOL,
        EXPECTED_PROTOCOL_SHA256,
    ),

    (
        "correction worker",
        CORRECTION_WORKER,
        EXPECTED_WORKER_SHA256,
    ),

    (
        "erratum record",
        ERRATUM_RECORD,
        EXPECTED_ERRATUM_SHA256,
    ),

    (
        "freeze receipt",
        FREEZE_RECEIPT,
        EXPECTED_FREEZE_RECEIPT_SHA256,
    ),

    (
        "lock manifest",
        LOCK_MANIFEST,
        EXPECTED_LOCK_MANIFEST_SHA256,
    ),

    (
        "warm raw",
        WARM_RAW,
        EXPECTED_WARM_RAW_SHA256,
    ),

    (
        "historical summary",
        WARM_SUMMARY,
        EXPECTED_WARM_SUMMARY_SHA256,
    ),

    (
        "condition status",
        WARM_STATUS,
        EXPECTED_WARM_STATUS_SHA256,
    ),

    (
        "Stage26-6D frontier",
        FRONTIER_6D,
        EXPECTED_FRONTIER_6D_SHA256,
    ),
]


for label, path, expected in checks:

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:28s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Identity mismatch: {label}"
        )


# =============================================================================
# 5. LOCKED SCIENTIFIC SEMANTICS GATE
# =============================================================================

banner(
    "STAGE26-6F1 :: LOCKED SCIENTIFIC SEMANTICS"
)


protocol = json.loads(
    CORRECTION_PROTOCOL.read_text(
        encoding="utf-8"
    )
)

freeze = json.loads(
    FREEZE_RECEIPT.read_text(
        encoding="utf-8"
    )
)


if protocol[
    "status"
] != "FROZEN_BEFORE_CORRECTED_CI_COMPUTATION":

    raise RuntimeError(
        "Correction was not frozen before execution."
    )


if protocol[
    "bootstrap"
][
    "replicates"
] != 2000:

    raise RuntimeError(
        "Frozen bootstrap replicate count changed."
    )


if protocol[
    "bootstrap"
][
    "literal_seed"
] != 26042:

    raise RuntimeError(
        "Frozen bootstrap seed changed."
    )


if protocol[
    "bootstrap"
][
    "derived_seed_allowed"
] is not False:

    raise RuntimeError(
        "Derived bootstrap seeds unexpectedly allowed."
    )


if freeze[
    "scientific_state"
][
    "bootstrap_executed"
] is not False:

    raise RuntimeError(
        "6F0 says bootstrap was already executed."
    )


if freeze[
    "scientific_state"
][
    "corrected_CI_computed"
] is not False:

    raise RuntimeError(
        "6F0 says corrected CI was already computed."
    )


if freeze[
    "scientific_state"
][
    "Pareto_frontier_changed"
] is not False:

    raise RuntimeError(
        "Pareto frontier changed before correction."
    )


print(
    "Replicates                : 2,000"
)

print(
    "RNG                       : numpy.random.default_rng / PCG64"
)

print(
    "Literal seed              : 26042"
)

print(
    "Derived seeds             : FORBIDDEN"
)

print(
    "Condition-local seed reset: YES"
)

print(
    "Same resample indices     : YES"
)

print(
    "Latency unit              : milliseconds before percentile"
)

print(
    "Corrected values pre-seen : NO"
)

print(
    "Pareto membership mutable : NO"
)


# =============================================================================
# 6. ENVIRONMENT / RNG IMPLEMENTATION GATE
# =============================================================================

banner(
    "STAGE26-6F1 :: DERIVED-STATISTICS ENVIRONMENT"
)


rng_probe = np.random.default_rng(
    26042
)


print(
    "Python:",
    sys.version.split()[0]
)

print(
    "NumPy :",
    np.__version__
)

print(
    "default_rng bit generator:",
    type(
        rng_probe.bit_generator
    ).__name__
)


if type(
    rng_probe.bit_generator
).__name__ != "PCG64":

    raise RuntimeError(
        "np.random.default_rng is not using PCG64."
    )


del rng_probe


# =============================================================================
# 7. ORIGINAL STAGE26-2 STATUS INVENTORY
# =============================================================================

banner(
    "STAGE26-6F1 :: ORIGINAL CONDITION INVENTORY"
)


status_rows = load_csv(
    WARM_STATUS
)


if len(
    status_rows
) != 80:

    raise RuntimeError(
        f"Expected 80 Stage26-2 conditions; found {len(status_rows)}."
    )


original_status_counts = Counter(
    row[
        "status"
    ]
    for row in status_rows
)


print(
    "Condition count:",
    len(
        status_rows
    )
)

print(
    "Status counts:"
)

for key in sorted(
    original_status_counts
):

    print(
        f"  {key:10s}: "
        f"{original_status_counts[key]}"
    )


if original_status_counts.get(
    "PASS",
    0,
) != 73:

    raise RuntimeError(
        "Expected 73 PASS Stage26-2 conditions."
    )


# =============================================================================
# 8. CREATE OUTPUT DIRECTORY
# =============================================================================

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# =============================================================================
# 9. EXECUTE EXACT LOCKED WORKER — ONCE
# =============================================================================

banner(
    "STAGE26-6F1 :: EXECUTE LOCKED CORRECTION WORKER"
)


worker_sha_immediately_before = sha256_file(
    CORRECTION_WORKER
)


if worker_sha_immediately_before != EXPECTED_WORKER_SHA256:

    raise RuntimeError(
        "Locked worker changed immediately before execution."
    )


worker_cmd = [
    sys.executable,
    str(
        CORRECTION_WORKER
    ),
    "--raw-csv",
    str(
        WARM_RAW
    ),
    "--summary-csv",
    str(
        WARM_SUMMARY
    ),
    "--condition-status-csv",
    str(
        WARM_STATUS
    ),
    "--output-json",
    str(
        CORRECTED_JSON
    ),
]


print(
    "Worker SHA256:",
    worker_sha_immediately_before
)

print(
    "Invocation:"
)

print(
    " ",
    " ".join(
        worker_cmd
    )
)


worker_run = run(
    worker_cmd,
    cwd=REPO,
    check=False,
    text=True,
)


print(
    "\nReturn code:",
    worker_run.returncode
)


if worker_run.stdout.strip():

    print(
        "\nWorker output:"
    )

    print(
        worker_run.stdout.strip()
    )


if worker_run.returncode != 0:

    raise RuntimeError(
        "Locked Stage26-6F1 correction worker failed.\n"
        +
        worker_run.stdout
    )


if not CORRECTED_JSON.is_file():

    raise RuntimeError(
        "Locked worker returned success but corrected JSON is absent."
    )


worker_sha_after = sha256_file(
    CORRECTION_WORKER
)


if worker_sha_after != EXPECTED_WORKER_SHA256:

    raise RuntimeError(
        "Locked worker changed during execution."
    )


print(
    "\nLocked worker execution: PASS"
)


# =============================================================================
# 10. VERIFY IMMUTABLE INPUTS REMAIN BYTE-IDENTICAL
# =============================================================================

banner(
    "STAGE26-6F1 :: POST-EXECUTION IMMUTABILITY GATE"
)


post_input_checks = [
    (
        "warm raw",
        WARM_RAW,
        EXPECTED_WARM_RAW_SHA256,
    ),

    (
        "historical summary",
        WARM_SUMMARY,
        EXPECTED_WARM_SUMMARY_SHA256,
    ),

    (
        "condition status",
        WARM_STATUS,
        EXPECTED_WARM_STATUS_SHA256,
    ),

    (
        "6F0 worker",
        CORRECTION_WORKER,
        EXPECTED_WORKER_SHA256,
    ),

    (
        "6D frontier",
        FRONTIER_6D,
        EXPECTED_FRONTIER_6D_SHA256,
    ),
]


for label, path, expected in post_input_checks:

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:24s}: "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Immutable source changed during 6F1: {label}"
        )


# =============================================================================
# 11. LOAD / AUDIT CORRECTED OUTPUT
# =============================================================================

banner(
    "STAGE26-6F1 :: CORRECTED OUTPUT SCIENTIFIC AUDIT"
)


corrected = json.loads(
    CORRECTED_JSON.read_text(
        encoding="utf-8"
    )
)


bp = corrected[
    "bootstrap_protocol"
]


expected_bp = {
    "ci_percentiles": [
        2.5,
        97.5,
    ],

    "condition_local_seed_reset":
        True,

    "percentile_method":
        "linear",

    "replicates":
        2000,

    "rng":
        "numpy.random.default_rng_PCG64",

    "same_resample_indices_for_all_targets":
        True,

    "seed":
        26042,
}


if bp != expected_bp:

    raise RuntimeError(
        "Executed bootstrap protocol differs from frozen expectation."
    )


conditions = corrected[
    "conditions"
]


if len(
    conditions
) != 80:

    raise RuntimeError(
        f"Corrected result must contain 80 conditions; found {len(conditions)}."
    )


corrected_status_counts = Counter(
    row[
        "status"
    ]
    for row in conditions
)


print(
    "Corrected condition count:",
    len(
        conditions
    )
)

print(
    "Corrected status counts:"
)


for key in sorted(
    corrected_status_counts
):

    print(
        f"  {key:10s}: "
        f"{corrected_status_counts[key]}"
    )


if corrected_status_counts != original_status_counts:

    raise RuntimeError(
        "Corrected output changed Stage26-2 condition outcomes."
    )


pass_rows = [
    row
    for row in conditions
    if row[
        "status"
    ]
    ==
    "PASS"
]


nonpass_rows = [
    row
    for row in conditions
    if row[
        "status"
    ]
    !=
    "PASS"
]


if len(
    pass_rows
) != 73:

    raise RuntimeError(
        "Corrected PASS count is not 73."
    )


if any(
    row[
        "bootstrap_computed"
    ]
    is not True
    for row in pass_rows
):

    raise RuntimeError(
        "A PASS condition lacks corrected bootstrap uncertainty."
    )


if any(
    row[
        "bootstrap_computed"
    ]
    is not False
    for row in nonpass_rows
):

    raise RuntimeError(
        "Bootstrap was fabricated for a non-PASS condition."
    )


print(
    "\nPASS conditions bootstrapped    : 73/73"
)

print(
    "Non-PASS conditions bootstrapped: 0"
)


# =============================================================================
# 12. EXACT POINT-ESTIMATE INTEGRITY AGAINST HISTORICAL SUMMARY
# =============================================================================

banner(
    "STAGE26-6F1 :: POINT-ESTIMATE IMMUTABILITY"
)


summary_rows = load_csv(
    WARM_SUMMARY
)

summary_by_condition = {
    row[
        "condition_id"
    ]:
        row
    for row in summary_rows
}


point_integrity_count = 0


for row in pass_rows:

    condition_id = row[
        "condition_id"
    ]

    historical = summary_by_condition[
        condition_id
    ]

    point = row[
        "point_estimate_integrity"
    ]


    checks = [
        (
            float(
                point[
                    "p50_batch_latency_ms"
                ]
            )
            ==
            float(
                historical[
                    "p50_batch_latency_ms"
                ]
            )
        ),

        (
            float(
                point[
                    "p95_batch_latency_ms"
                ]
            )
            ==
            float(
                historical[
                    "p95_batch_latency_ms"
                ]
            )
        ),

        (
            float(
                point[
                    "median_throughput_flows_per_second"
                ]
            )
            ==
            float(
                historical[
                    "median_throughput_flows_per_second"
                ]
            )
        ),
    ]


    n = int(
        row[
            "n"
        ]
    )


    if n >= 100:

        checks.append(
            float(
                point[
                    "p99_batch_latency_ms_if_n_gte_100"
                ]
            )
            ==
            float(
                historical[
                    "p99_batch_latency_ms_if_n_gte_100"
                ]
            )
        )


    else:

        checks.append(
            point[
                "p99_batch_latency_ms_if_n_gte_100"
            ]
            is None
        )


    if not all(
        checks
    ):

        raise RuntimeError(
            f"{condition_id}: point-estimate immutability failed."
        )


    point_integrity_count += 1


print(
    "Exact PASS-condition point estimates verified:",
    point_integrity_count,
    "/ 73"
)


# =============================================================================
# 13. VALIDATE CORRECTED CI GEOMETRY
# =============================================================================

banner(
    "STAGE26-6F1 :: CORRECTED CI GEOMETRY"
)


for row in pass_rows:

    condition_id = row[
        "condition_id"
    ]

    point = row[
        "point_estimate_integrity"
    ]

    ci = row[
        "corrected_ci95"
    ]


    for metric in [
        "p50_batch_latency_ms",
        "p95_batch_latency_ms",
        "median_throughput_flows_per_second",
    ]:

        bounds = ci[
            metric
        ]


        if (
            not isinstance(
                bounds,
                list,
            )
            or
            len(
                bounds
            )
            !=
            2
        ):

            raise RuntimeError(
                f"{condition_id}: malformed corrected CI for {metric}."
            )


        low = float(
            bounds[
                0
            ]
        )

        high = float(
            bounds[
                1
            ]
        )

        estimate = float(
            point[
                metric
            ]
        )


        if low > high:

            raise RuntimeError(
                f"{condition_id}: inverted corrected CI for {metric}."
            )


        if not (
            low
            <=
            estimate
            <=
            high
        ):

            raise RuntimeError(
                f"{condition_id}: point estimate outside corrected CI "
                f"for {metric}."
            )


    p99_ci = ci[
        "p99_batch_latency_ms_if_n_gte_100"
    ]


    if int(
        row[
            "n"
        ]
    ) >= 100:

        if (
            not isinstance(
                p99_ci,
                list,
            )
            or
            len(
                p99_ci
            )
            !=
            2
        ):

            raise RuntimeError(
                f"{condition_id}: expected p99 CI for n>=100."
            )


        p99_point = float(
            point[
                "p99_batch_latency_ms_if_n_gte_100"
            ]
        )


        if not (
            float(
                p99_ci[
                    0
                ]
            )
            <=
            p99_point
            <=
            float(
                p99_ci[
                    1
                ]
            )
        ):

            raise RuntimeError(
                f"{condition_id}: p99 point outside corrected CI."
            )


    else:

        if p99_ci is not None:

            raise RuntimeError(
                f"{condition_id}: p99 CI exists despite n<100."
            )


print(
    "Corrected CI structural/containment audit: PASS"
)


# =============================================================================
# 14. WRITE PUBLICATION-FACING CORRECTED CONDITION TABLE
# =============================================================================

banner(
    "STAGE26-6F1 :: WRITE CORRECTED CONDITION TABLE"
)


condition_fields = [
    "condition_id",
    "target_id",
    "comparison_group",
    "hardware_mode",
    "thread_count",
    "batch_size",
    "status",
    "n",

    "p50_point_ms",
    "historical_p50_ci_low_ms",
    "historical_p50_ci_high_ms",
    "corrected_p50_ci_low_ms",
    "corrected_p50_ci_high_ms",

    "p95_point_ms",
    "historical_p95_ci_low_ms",
    "historical_p95_ci_high_ms",
    "corrected_p95_ci_low_ms",
    "corrected_p95_ci_high_ms",

    "p99_point_ms_if_n_gte_100",
    "historical_p99_ci_low_ms_if_n_gte_100",
    "historical_p99_ci_high_ms_if_n_gte_100",
    "corrected_p99_ci_low_ms_if_n_gte_100",
    "corrected_p99_ci_high_ms_if_n_gte_100",

    "median_throughput_point",
    "historical_median_throughput_ci_low",
    "historical_median_throughput_ci_high",
    "corrected_median_throughput_ci_low",
    "corrected_median_throughput_ci_high",

    "historical_ci_status",
    "corrected_ci_status",
]


with CORRECTED_CSV.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=condition_fields,
    )

    writer.writeheader()


    for row in conditions:

        if row[
            "status"
        ] != "PASS":

            writer.writerow(
                {
                    "condition_id":
                        row[
                            "condition_id"
                        ],

                    "status":
                        row[
                            "status"
                        ],

                    "historical_ci_status":
                        "NOT_APPLICABLE_NON_PASS",

                    "corrected_ci_status":
                        "NOT_APPLICABLE_NON_PASS",
                }
            )

            continue


        point = row[
            "point_estimate_integrity"
        ]

        old = row[
            "historical_ci95"
        ]

        new = row[
            "corrected_ci95"
        ]


        old_p99 = old[
            "p99_batch_latency_ms_if_n_gte_100"
        ]

        new_p99 = new[
            "p99_batch_latency_ms_if_n_gte_100"
        ]


        writer.writerow(
            {
                "condition_id":
                    row[
                        "condition_id"
                    ],

                "target_id":
                    row[
                        "target_id"
                    ],

                "comparison_group":
                    row[
                        "comparison_group"
                    ],

                "hardware_mode":
                    row[
                        "hardware_mode"
                    ],

                "thread_count":
                    row[
                        "thread_count"
                    ],

                "batch_size":
                    row[
                        "batch_size"
                    ],

                "status":
                    row[
                        "status"
                    ],

                "n":
                    row[
                        "n"
                    ],

                "p50_point_ms":
                    point[
                        "p50_batch_latency_ms"
                    ],

                "historical_p50_ci_low_ms":
                    old[
                        "p50_batch_latency_ms"
                    ][
                        0
                    ],

                "historical_p50_ci_high_ms":
                    old[
                        "p50_batch_latency_ms"
                    ][
                        1
                    ],

                "corrected_p50_ci_low_ms":
                    new[
                        "p50_batch_latency_ms"
                    ][
                        0
                    ],

                "corrected_p50_ci_high_ms":
                    new[
                        "p50_batch_latency_ms"
                    ][
                        1
                    ],

                "p95_point_ms":
                    point[
                        "p95_batch_latency_ms"
                    ],

                "historical_p95_ci_low_ms":
                    old[
                        "p95_batch_latency_ms"
                    ][
                        0
                    ],

                "historical_p95_ci_high_ms":
                    old[
                        "p95_batch_latency_ms"
                    ][
                        1
                    ],

                "corrected_p95_ci_low_ms":
                    new[
                        "p95_batch_latency_ms"
                    ][
                        0
                    ],

                "corrected_p95_ci_high_ms":
                    new[
                        "p95_batch_latency_ms"
                    ][
                        1
                    ],

                "p99_point_ms_if_n_gte_100":
                    point[
                        "p99_batch_latency_ms_if_n_gte_100"
                    ],

                "historical_p99_ci_low_ms_if_n_gte_100":
                    (
                        old_p99[
                            0
                        ]
                        if old_p99 is not None
                        else ""
                    ),

                "historical_p99_ci_high_ms_if_n_gte_100":
                    (
                        old_p99[
                            1
                        ]
                        if old_p99 is not None
                        else ""
                    ),

                "corrected_p99_ci_low_ms_if_n_gte_100":
                    (
                        new_p99[
                            0
                        ]
                        if new_p99 is not None
                        else ""
                    ),

                "corrected_p99_ci_high_ms_if_n_gte_100":
                    (
                        new_p99[
                            1
                        ]
                        if new_p99 is not None
                        else ""
                    ),

                "median_throughput_point":
                    point[
                        "median_throughput_flows_per_second"
                    ],

                "historical_median_throughput_ci_low":
                    old[
                        "median_throughput_flows_per_second"
                    ][
                        0
                    ],

                "historical_median_throughput_ci_high":
                    old[
                        "median_throughput_flows_per_second"
                    ][
                        1
                    ],

                "corrected_median_throughput_ci_low":
                    new[
                        "median_throughput_flows_per_second"
                    ][
                        0
                    ],

                "corrected_median_throughput_ci_high":
                    new[
                        "median_throughput_flows_per_second"
                    ][
                        1
                    ],

                "historical_ci_status":
                    row[
                        "historical_CI_status"
                    ],

                "corrected_ci_status":
                    row[
                        "corrected_CI_status"
                    ],
            }
        )


# =============================================================================
# 15. SIX-POINT PARETO LATENCY UNCERTAINTY TABLE
# =============================================================================

banner(
    "STAGE26-6F1 :: SIX-POINT P95 UNCERTAINTY"
)


frontier = json.loads(
    FRONTIER_6D.read_text(
        encoding="utf-8"
    )
)


frontier_members = {
    point[
        "target_id"
    ]:
        bool(
            point[
                "frontier_member"
            ]
        )
    for point in frontier[
        "points"
    ]
}


conditions_by_id = {
    row[
        "condition_id"
    ]:
        row
    for row in conditions
}


pareto_fields = [
    "target_id",
    "comparison_group",
    "condition_id",
    "p95_point_ms",
    "historical_ci95_low_ms",
    "historical_ci95_high_ms",
    "corrected_ci95_low_ms",
    "corrected_ci95_high_ms",
    "corrected_ci_width_ms",
    "historical_low_minus_corrected_low_ms",
    "historical_high_minus_corrected_high_ms",
    "stage26_6d_frontier_member",
    "frontier_membership_changed_by_uncertainty",
    "claim_boundary",
]


pareto_output_rows = []


for target, condition_id in PARETO_CONDITIONS.items():

    row = conditions_by_id[
        condition_id
    ]


    if row[
        "target_id"
    ] != target:

        raise RuntimeError(
            f"{condition_id}: Pareto target mapping mismatch."
        )


    if row[
        "status"
    ] != "PASS":

        raise RuntimeError(
            f"{condition_id}: frozen Pareto condition is not PASS."
        )


    point = float(
        row[
            "point_estimate_integrity"
        ][
            "p95_batch_latency_ms"
        ]
    )

    historical_low = float(
        row[
            "historical_ci95"
        ][
            "p95_batch_latency_ms"
        ][
            0
        ]
    )

    historical_high = float(
        row[
            "historical_ci95"
        ][
            "p95_batch_latency_ms"
        ][
            1
        ]
    )

    corrected_low = float(
        row[
            "corrected_ci95"
        ][
            "p95_batch_latency_ms"
        ][
            0
        ]
    )

    corrected_high = float(
        row[
            "corrected_ci95"
        ][
            "p95_batch_latency_ms"
        ][
            1
        ]
    )


    claim_boundary = (
        "DESCRIPTIVE_NON_CONFIRMATORY"
        if row[
            "comparison_group"
        ]
        ==
        "GROUP_B_PACKET_IMAGE"
        else
        "PRIMARY_WITHIN_GROUP_POINT_ESTIMATE"
    )


    record = {
        "target_id":
            target,

        "comparison_group":
            row[
                "comparison_group"
            ],

        "condition_id":
            condition_id,

        "p95_point_ms":
            point,

        "historical_ci95_low_ms":
            historical_low,

        "historical_ci95_high_ms":
            historical_high,

        "corrected_ci95_low_ms":
            corrected_low,

        "corrected_ci95_high_ms":
            corrected_high,

        "corrected_ci_width_ms":
            (
                corrected_high
                -
                corrected_low
            ),

        "historical_low_minus_corrected_low_ms":
            (
                historical_low
                -
                corrected_low
            ),

        "historical_high_minus_corrected_high_ms":
            (
                historical_high
                -
                corrected_high
            ),

        "stage26_6d_frontier_member":
            frontier_members[
                target
            ],

        "frontier_membership_changed_by_uncertainty":
            False,

        "claim_boundary":
            claim_boundary,
    }


    pareto_output_rows.append(
        record
    )


    print(
        f"{target:42s} "
        f"p95={point:12.9f} "
        f"historical=[{historical_low:12.9f}, {historical_high:12.9f}] "
        f"corrected=[{corrected_low:12.9f}, {corrected_high:12.9f}] "
        f"frontier={'YES' if frontier_members[target] else 'NO'}"
    )


with PARETO_CSV.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=pareto_fields,
    )

    writer.writeheader()

    writer.writerows(
        pareto_output_rows
    )


# =============================================================================
# 16. HISTORICAL VS CORRECTED P95 CI CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-6F1 :: HISTORICAL VS CORRECTED P95 CI AUDIT"
)


p95_changes = []


for row in pass_rows:

    old_low, old_high = row[
        "historical_ci95"
    ][
        "p95_batch_latency_ms"
    ]

    new_low, new_high = row[
        "corrected_ci95"
    ][
        "p95_batch_latency_ms"
    ]


    p95_changes.append(
        {
            "condition_id":
                row[
                    "condition_id"
                ],

            "low_abs_change_ms":
                abs(
                    float(
                        new_low
                    )
                    -
                    float(
                        old_low
                    )
                ),

            "high_abs_change_ms":
                abs(
                    float(
                        new_high
                    )
                    -
                    float(
                        old_high
                    )
                ),

            "exact_same_interval":
                (
                    float(
                        new_low
                    )
                    ==
                    float(
                        old_low
                    )
                    and
                    float(
                        new_high
                    )
                    ==
                    float(
                        old_high
                    )
                ),
        }
    )


exact_same_count = sum(
    1
    for row in p95_changes
    if row[
        "exact_same_interval"
    ]
)


max_low_change = max(
    row[
        "low_abs_change_ms"
    ]
    for row in p95_changes
)

max_high_change = max(
    row[
        "high_abs_change_ms"
    ]
    for row in p95_changes
)


print(
    "PASS conditions:",
    len(
        p95_changes
    )
)

print(
    "Historical p95 CI exactly reproduced:",
    exact_same_count
)

print(
    "Historical p95 CI changed:",
    len(
        p95_changes
    )
    -
    exact_same_count
)

print(
    "Maximum |lower-bound change| ms:",
    repr(
        max_low_change
    )
)

print(
    "Maximum |upper-bound change| ms:",
    repr(
        max_high_change
    )
)


# =============================================================================
# 17. EXECUTION RECEIPT
# =============================================================================

corrected_json_sha = sha256_file(
    CORRECTED_JSON
)

corrected_csv_sha = sha256_file(
    CORRECTED_CSV
)

pareto_csv_sha = sha256_file(
    PARETO_CSV
)


execution_receipt = {
    "schema":
        "stage26_6f1_correction_execution_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-6F1",

    "status":
        "PASS_PROTOCOL_CERTIFIED_DERIVED_TIMING_UNCERTAINTY",

    "scientific_parent":
        EXPECTED_PARENT,

    "locked_implementation": {
        "protocol_sha256":
            EXPECTED_PROTOCOL_SHA256,

        "worker_sha256":
            EXPECTED_WORKER_SHA256,

        "freeze_receipt_sha256":
            EXPECTED_FREEZE_RECEIPT_SHA256,

        "lock_manifest_sha256":
            EXPECTED_LOCK_MANIFEST_SHA256,
    },

    "immutable_inputs": {
        "warm_raw_sha256":
            EXPECTED_WARM_RAW_SHA256,

        "historical_summary_sha256":
            EXPECTED_WARM_SUMMARY_SHA256,

        "condition_status_sha256":
            EXPECTED_WARM_STATUS_SHA256,

        "stage26_6d_frontier_sha256":
            EXPECTED_FRONTIER_6D_SHA256,
    },

    "execution_environment": {
        "python":
            sys.version,

        "numpy":
            np.__version__,

        "default_rng_bit_generator":
            "PCG64",
    },

    "bootstrap": {
        "replicates":
            2000,

        "seed":
            26042,

        "rng":
            "numpy.random.default_rng_PCG64",

        "derived_seeds":
            False,

        "condition_local_seed_reset":
            True,

        "same_resample_indices_within_condition":
            True,

        "latency_unit_before_percentile":
            "milliseconds",

        "percentile_method":
            "linear",

        "CI_percentiles": [
            2.5,
            97.5,
        ],
    },

    "condition_outcomes": {
        "total":
            len(
                conditions
            ),

        "PASS":
            original_status_counts.get(
                "PASS",
                0,
            ),

        "non_PASS":
            len(
                conditions
            )
            -
            original_status_counts.get(
                "PASS",
                0,
            ),

        "status_counts":
            dict(
                original_status_counts
            ),

        "bootstrapped_conditions":
            len(
                pass_rows
            ),

        "fabricated_non_PASS_intervals":
            0,
    },

    "point_estimate_integrity": {
        "PASS_conditions_exactly_verified":
            point_integrity_count,

        "point_estimates_modified":
            False,
    },

    "historical_vs_corrected_p95": {
        "historical_CI_label":
            "HISTORICAL_NOT_PROTOCOL_CERTIFIED",

        "corrected_CI_label":
            "PROTOCOL_CERTIFIED_DERIVED_UNCERTAINTY",

        "conditions_with_exact_same_p95_interval":
            exact_same_count,

        "conditions_with_changed_p95_interval":
            (
                len(
                    p95_changes
                )
                -
                exact_same_count
            ),

        "maximum_absolute_lower_bound_change_ms":
            max_low_change,

        "maximum_absolute_upper_bound_change_ms":
            max_high_change,
    },

    "six_point_pareto_uncertainty": {
        "condition_count":
            6,

        "frontier_membership_recomputed":
            False,

        "frontier_membership_changed":
            False,

        "PR_AUC_bootstrap_computed":
            False,
    },

    "outputs": {
        "corrected_json_sha256":
            corrected_json_sha,

        "corrected_condition_csv_sha256":
            corrected_csv_sha,

        "six_point_p95_csv_sha256":
            pareto_csv_sha,
    },

    "scientific_state": {
        "bootstrap_executed":
            True,

        "corrected_CI_computed":
            True,

        "historical_CI_overwritten":
            False,

        "raw_measurement_modified":
            False,

        "point_estimate_modified":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "predictive_metric_recomputed":
            False,

        "PR_AUC_bootstrap_computed":
            False,

        "holdout_reopened":
            False,

        "Pareto_frontier_recomputed":
            False,

        "Pareto_frontier_changed":
            False,

        "cross_group_frontier_computed":
            False,

        "PCAP_accessed":
            False,

        "Release_corpus_accessed":
            False,

        "GPU_used":
            False,
    },

    "publication_policy": {
        "use_historical_stage26_2_CIs":
            False,

        "use_stage26_6f1_corrected_timing_CIs":
            True,

        "retain_historical_CIs_for_audit":
            True,

        "stage26_6d_point_estimate_frontier_remains_authoritative":
            True,
    },

    "next":
        (
            "Proceed to Stage26-7 CPU batch/component analysis using the "
            "immutable Stage26 measurements and the corrected Stage26-6F1 "
            "timing uncertainty where uncertainty reporting is required."
        ),
}


atomic_json(
    EXECUTION_RECEIPT,
    execution_receipt,
)


receipt_sha = sha256_file(
    EXECUTION_RECEIPT
)


# =============================================================================
# 18. PACKAGE MANIFEST
# =============================================================================

package_files = [
    CORRECTED_JSON,
    CORRECTED_CSV,
    PARETO_CSV,
    EXECUTION_RECEIPT,
]


manifest_rows = []


for path in package_files:

    manifest_rows.append(
        {
            "repo_relative_path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest = {
    "schema":
        "stage26_6f1_corrected_uncertainty_manifest_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-6F1",

    "status":
        "READY_FOR_GIT_ANCHOR",

    "scientific_parent":
        EXPECTED_PARENT,

    "commit_subject":
        COMMIT_SUBJECT,

    "locked_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "locked_worker_sha256":
        EXPECTED_WORKER_SHA256,

    "immutable_warm_raw_sha256":
        EXPECTED_WARM_RAW_SHA256,

    "immutable_historical_summary_sha256":
        EXPECTED_WARM_SUMMARY_SHA256,

    "stage26_6d_frontier_sha256":
        EXPECTED_FRONTIER_6D_SHA256,

    "corrected_json_sha256":
        corrected_json_sha,

    "corrected_condition_csv_sha256":
        corrected_csv_sha,

    "six_point_p95_csv_sha256":
        pareto_csv_sha,

    "execution_receipt_sha256":
        receipt_sha,

    "file_count_excluding_manifest":
        len(
            manifest_rows
        ),

    "files":
        manifest_rows,

    "scientific_state": {
        "corrected_uncertainty_computed":
            True,

        "new_measurement_performed":
            False,

        "historical_artifact_overwritten":
            False,

        "point_estimate_changed":
            False,

        "Pareto_frontier_changed":
            False,

        "predictive_bootstrap_performed":
            False,

        "GPU_used":
            False,
    },
}


atomic_json(
    MANIFEST,
    manifest,
)


manifest_sha = sha256_file(
    MANIFEST
)


# =============================================================================
# 19. LOCAL PACKAGE AUDIT
# =============================================================================

banner(
    "STAGE26-6F1 :: LOCAL PACKAGE AUDIT"
)


for row in manifest_rows:

    path = (
        REPO
        /
        row[
            "repo_relative_path"
        ]
    )

    size = int(
        path.stat().st_size
    )

    digest = sha256_file(
        path
    )

    passed = (
        size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        digest
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{size:10,d} B "
        f"{digest} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Local Stage26-6F1 package audit failed."
        )


print(
    "\nManifest SHA256:"
)

print(
    " ",
    manifest_sha
)


# =============================================================================
# 20. GIT CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-6F1 :: GIT CHANGE AUDIT"
)


repo_status = git(
    "status",
    "--porcelain",
)


print(
    repo_status
)


if not repo_status:

    raise RuntimeError(
        "Expected uncommitted Stage26-6F1 package."
    )


unexpected = []


for line in repo_status.splitlines():

    relpath = line[
        3:
    ]


    if not relpath.startswith(
        str(
            CHECKPOINT_REL
        )
        +
        "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository changes:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 21. GIT IDENTITY
# =============================================================================

author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_PARENT,
)

author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_PARENT,
)


if (
    not author_name.strip()
    or
    "@"
    not in
    author_email
):

    raise RuntimeError(
        "Could not recover Git identity."
    )


git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


print(
    "\nGit author:"
)

print(
    " ",
    author_name,
    "<" + author_email + ">"
)


# =============================================================================
# 22. COMMIT
# =============================================================================

banner(
    "STAGE26-6F1 :: COMMIT"
)


git(
    "add",
    str(
        CHECKPOINT_REL
    ),
)


print(
    git(
        "diff",
        "--cached",
        "--name-status",
    )
)


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-6F1 commit parent mismatch."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Stage26-6F1 commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository not clean after Stage26-6F1 commit."
    )


# =============================================================================
# 23. PUSH
# =============================================================================

banner(
    "STAGE26-6F1 :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    push_result = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
        text=True,
    )


    print(
        push_result.stdout.strip()
    )


github_token = None


# =============================================================================
# 24. REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-6F1 :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "origin/main did not advance to Stage26-6F1."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote Stage26-6F1 parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote Stage26-6F1 subject mismatch."
    )


# =============================================================================
# 25. REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-6F1 :: REMOTE BYTE VERIFICATION"
)


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    str(
        MANIFEST.relative_to(
            REPO
        )
    ),
)


remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "Remote manifest SHA256:"
)

print(
    " ",
    remote_manifest_sha
)


if remote_manifest_sha != manifest_sha:

    raise RuntimeError(
        "Remote Stage26-6F1 manifest mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


for row in remote_manifest[
    "files"
]:

    data = git_blob_bytes(
        "origin/main",
        row[
            "repo_relative_path"
        ],
    )

    actual_size = len(
        data
    )

    actual_sha = sha256_bytes(
        data
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Remote Stage26-6F1 byte verification failed."
        )


# =============================================================================
# 26. REMOTE SCIENTIFIC VERIFICATION
# =============================================================================

banner(
    "STAGE26-6F1 :: REMOTE SCIENTIFIC VERIFICATION"
)


remote_result = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            CORRECTED_JSON.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_receipt = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            EXECUTION_RECEIPT.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)


remote_conditions = remote_result[
    "conditions"
]


remote_pass = [
    row
    for row in remote_conditions
    if row[
        "status"
    ]
    ==
    "PASS"
]


remote_nonpass = [
    row
    for row in remote_conditions
    if row[
        "status"
    ]
    !=
    "PASS"
]


remote_checks = {
    "80 conditions":
        (
            len(
                remote_conditions
            )
            ==
            80
        ),

    "73 PASS":
        (
            len(
                remote_pass
            )
            ==
            73
        ),

    "PASS uncertainty complete":
        all(
            row[
                "bootstrap_computed"
            ]
            is True
            for row in remote_pass
        ),

    "non-PASS uncertainty absent":
        all(
            row[
                "bootstrap_computed"
            ]
            is False
            for row in remote_nonpass
        ),

    "2,000 replicates":
        (
            remote_result[
                "bootstrap_protocol"
            ][
                "replicates"
            ]
            ==
            2000
        ),

    "seed 26042":
        (
            remote_result[
                "bootstrap_protocol"
            ][
                "seed"
            ]
            ==
            26042
        ),

    "PCG64":
        (
            remote_result[
                "bootstrap_protocol"
            ][
                "rng"
            ]
            ==
            "numpy.random.default_rng_PCG64"
        ),

    "condition-local reset":
        (
            remote_result[
                "bootstrap_protocol"
            ][
                "condition_local_seed_reset"
            ]
            is True
        ),

    "same resample indices":
        (
            remote_result[
                "bootstrap_protocol"
            ][
                "same_resample_indices_for_all_targets"
            ]
            is True
        ),

    "correction PASS":
        (
            remote_receipt[
                "status"
            ]
            ==
            "PASS_PROTOCOL_CERTIFIED_DERIVED_TIMING_UNCERTAINTY"
        ),

    "historical overwrite false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "historical_CI_overwritten"
            ]
            is False
        ),

    "point estimate changed false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "point_estimate_modified"
            ]
            is False
        ),

    "new inference false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "inference_performed"
            ]
            is False
        ),

    "new timing false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "timing_performed"
            ]
            is False
        ),

    "PR-AUC bootstrap false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "PR_AUC_bootstrap_computed"
            ]
            is False
        ),

    "Pareto recompute false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "Pareto_frontier_recomputed"
            ]
            is False
        ),

    "Pareto change false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "Pareto_frontier_changed"
            ]
            is False
        ),

    "GPU false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "GPU_used"
            ]
            is False
        ),
}


for name, passed in remote_checks.items():

    print(
        f"{name:38s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    remote_checks.values()
):

    raise RuntimeError(
        "Remote Stage26-6F1 scientific verification failed."
    )


# =============================================================================
# 27. FINAL CLOSURE
# =============================================================================

banner(
    "STAGE26-6F1 CORRECTED CPU TIMING UNCERTAINTY COMPLETE"
)


final_status = git(
    "status",
    "--porcelain",
)


print(
    "NEW DURABLE COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nPARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nCORRECTION STATUS:"
)

print(
    "  historical Stage26-2 CIs : retained, NOT protocol-certified"
)

print(
    "  Stage26-6F1 CIs          : protocol-certified derived uncertainty"
)

print(
    "  PASS conditions corrected:",
    len(
        pass_rows
    )
)

print(
    "  non-PASS intervals       : none fabricated"
)


print(
    "\nSIX PARETO CPU1/B1 P95 INTERVALS:"
)


for row in pareto_output_rows:

    print(
        f"  {row['target_id']:42s} "
        f"{row['p95_point_ms']:.9f} ms "
        f"[{row['corrected_ci95_low_ms']:.9f}, "
        f"{row['corrected_ci95_high_ms']:.9f}] "
        f"frontier={'YES' if row['stage26_6d_frontier_member'] else 'NO'}"
    )


print(
    "\nHISTORICAL-vs-CORRECTED P95 CI:"
)

print(
    "  exact same intervals:",
    exact_same_count
)

print(
    "  changed intervals   :",
    len(
        p95_changes
    )
    -
    exact_same_count
)

print(
    "  max lower change ms :",
    repr(
        max_low_change
    )
)

print(
    "  max upper change ms :",
    repr(
        max_high_change
    )
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  bootstrap correction executed    : YES"
)

print(
    "  protocol-certified timing CIs     : YES"
)

print(
    "  raw measurements changed          : NO"
)

print(
    "  point estimates changed           : NO"
)

print(
    "  historical Stage26-2 overwritten  : NO"
)

print(
    "  model loading                     : NO"
)

print(
    "  inference                         : NO"
)

print(
    "  timing remeasurement              : NO"
)

print(
    "  predictive metric recomputation   : NO"
)

print(
    "  PR-AUC bootstrap                  : NO"
)

print(
    "  Pareto frontier recomputed        : NO"
)

print(
    "  Pareto frontier changed           : NO"
)

print(
    "  cross-group frontier              : NO"
)

print(
    "  GPU                               : NO"
)


print(
    "\nHASHES:"
)

print(
    "  corrected JSON :",
    corrected_json_sha
)

print(
    "  condition CSV  :",
    corrected_csv_sha
)

print(
    "  six-point CSV  :",
    pareto_csv_sha
)

print(
    "  receipt        :",
    receipt_sha
)

print(
    "  manifest       :",
    manifest_sha
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  commit             : PASS"
)

print(
    "  parent             : PASS"
)

print(
    "  package bytes      : PASS"
)

print(
    "  corrected results  : PASS"
)

print(
    "  scientific content : PASS"
)


print(
    "\nRepo clean:",
    final_status == ""
)


if final_status:

    raise RuntimeError(
        "Repository not clean after Stage26-6F1."
    )


print(
    "\nNEXT:"
)

print(
    "  Stage26-7: CPU batch-scaling and scientifically compatible"
)

print(
    "  component-level bottleneck/capacity analysis."
)

print(
    "  Do NOT construct a complete raw-PCAP-to-model bottleneck ratio"
)

print(
    "  across missing Stage26-5 boundaries."
)

print(
    "  Stage26-6D Pareto membership remains immutable."
)

print(
    "  No GPU yet."
)


STAGE26-6F1 :: DURABLE SCIENTIFIC PARENT
Expected parent: 94a2c5c290f67e3f9dda10a049e6a5318e912bdd
Local HEAD     : 94a2c5c290f67e3f9dda10a049e6a5318e912bdd
origin/main    : 94a2c5c290f67e3f9dda10a049e6a5318e912bdd
Repo clean     : True

STAGE26-6F1 :: LOCKED IMPLEMENTATION / INPUT IDENTITY
correction protocol          PASS 9e267bf526304ce2c29ed3a74e8f2be29f0320de613860a3ea47752a9f9192f8
correction worker            PASS 9f4e6b6adb9a020128a3d935d7fc4e37412fa3b944d5d1ac7119c716a9cdc91b
erratum record               PASS 59dc41df1f805180697a04a6be60c2e36d193d5623a02700713e3b5827a437f3
freeze receipt               PASS bee8b8922497ba23f8ec73e46ccfd4c863baad869b4dbfca90f081ce701b77c2
lock manifest                PASS bb1509d721c8e26b14514d27790b534011ffac3f2452c712859a5f2dd4921c01
warm raw                     PASS 78c58289ccfc4598966d6516201028f43c4b1d16d8bd0268a0cf58129d4179fa
historical summary           PASS 75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232
condition stat

In [23]:
# =============================================================================
# STAGE26-7A
# READ-ONLY CPU CAPACITY / BATCH-SCALING SOURCE AND COMPATIBILITY INVENTORY
#
# DURABLE SCIENTIFIC PARENT:
#   a484148cd8ea7d7c89604d3a015f73bd6f1bc81a
#
# PURPOSE
# -------
# Resolve the exact committed sources that Stage26-7 is allowed to use and
# freeze, prospectively, which capacity/bottleneck comparisons are:
#
#   ALLOWED
#   DESCRIPTIVE_ONLY
#   PROHIBITED
#
# BEFORE any ratios or scaling conclusions are calculated.
#
# IMPORTANT STAGE26-5 BOUNDARY
# ----------------------------
#
# Group A complete E2E:
#   NOT AVAILABLE
#   Missing frozen 70-feature extraction path.
#
# Group B complete E2E:
#   NOT AVAILABLE
#   Missing raw-flow -> compact packet-image corpus bridge.
#
# Therefore:
#
#   raw extraction throughput
#       /
#   model inference throughput
#
# MUST NOT be called an end-to-end bottleneck ratio.
#
# SCIENTIFICALLY PERMITTED STAGE26-7 ANALYSES
# --------------------------------------------
#
# A. INFERENCE BATCH SCALING
#    - all frozen Stage26 inference targets
#    - within the same inference component only
#    - CPU1 and CPU2 may each be analyzed separately
#    - preserve OOM / TIMEOUT as resource-limit outcomes
#    - corrected Stage26-6F1 CIs used where uncertainty is reported
#
# B. GROUP-B REPRESENTATION-vs-INFERENCE COMPONENT CAPACITY
#    - CPU1 only
#    - identical matched batch size
#    - representation output unit: flows/images per second
#    - CNN/ViT inference input unit: flows/images per second
#    - only where BOTH conditions passed
#    - component-level ratio only
#    - NEVER complete raw-PCAP-to-model E2E
#
# C. RAW EXTRACTION
#    - may be reported as its own component throughput
#    - MUST NOT be algebraically composed across the missing 5A bridge
#
# D. GROUP-A EXTRACTION-vs-INFERENCE
#    - PROHIBITED as a pipeline bottleneck ratio
#    - frozen 70-feature extraction stage was never profiled
#
# E. CROSS-GROUP CAPACITY COMPARISON
#    - no predictive Pareto implications
#    - no complete-pipeline ranking
#
# THIS CELL DOES NOT:
#   - calculate any throughput ratio
#   - calculate speedup
#   - calculate scaling efficiency
#   - declare a bottleneck
#   - recompute Pareto
#   - perform bootstrap
#   - load models
#   - run inference
#   - run timing
#   - access PCAP
#   - access Release corpus
#   - use GPU
#   - modify Git
#
# OUTPUT:
#   TRANSIENT ONLY:
#
#   /kaggle/working/stage26_deployment_profiling/capacity/
#       stage26_7a_capacity_source_compatibility_inventory.json
# =============================================================================

from __future__ import annotations

import csv
import json
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "a484148cd8ea7d7c89604d3a015f73bd6f1bc81a"
)


RUNTIME_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

CAPACITY_RUNTIME = (
    RUNTIME_ROOT
    / "capacity"
)

OUT = (
    CAPACITY_RUNTIME
    / "stage26_7a_capacity_source_compatibility_inventory.json"
)


# -----------------------------------------------------------------------------
# Stage26-2 warm inference
# -----------------------------------------------------------------------------

WARM_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_2_cpu_warm_inference"
)

WARM_RAW = (
    WARM_DIR
    / "stage26_2_warm_raw.csv"
)

WARM_SUMMARY = (
    WARM_DIR
    / "stage26_2_warm_summary.csv"
)

WARM_STATUS = (
    WARM_DIR
    / "stage26_2_condition_status.csv"
)


EXPECTED_WARM_RAW_SHA256 = (
    "78c58289ccfc4598966d6516201028f43c4b1d16d8bd0268a0cf58129d4179fa"
)

EXPECTED_WARM_SUMMARY_SHA256 = (
    "75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232"
)

EXPECTED_WARM_STATUS_SHA256 = (
    "df146563826992cb57702e589e4cf5d875ecfdbbf79533a731405e8eb738e7af"
)


# -----------------------------------------------------------------------------
# Stage26-6F1 corrected uncertainty
# -----------------------------------------------------------------------------

CORRECTED_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_6f1_bootstrap_corrected_uncertainty"
)

CORRECTED_JSON = (
    CORRECTED_DIR
    / "stage26_6f1_bootstrap_corrected_uncertainty.json"
)

CORRECTED_CSV = (
    CORRECTED_DIR
    / "stage26_6f1_corrected_timing_uncertainty.csv"
)

CORRECTED_RECEIPT = (
    CORRECTED_DIR
    / "stage26_6f1_correction_execution_receipt.json"
)

CORRECTED_MANIFEST = (
    CORRECTED_DIR
    / "stage26_6f1_corrected_uncertainty_manifest.json"
)


EXPECTED_CORRECTED_JSON_SHA256 = (
    "45ea4408738fb5101ce68a363c2cc0a7951648eed49e474da9489b00ce6128ce"
)

EXPECTED_CORRECTED_CSV_SHA256 = (
    "f804f117312b967bfc13cc070a34be43545238bfd30a758a00ad58202c9e6828"
)

EXPECTED_CORRECTED_RECEIPT_SHA256 = (
    "8d37b6bc2e0a7a75b8102b7d1cd3c1c058907420b229c9da1f302d97a44f1130"
)

EXPECTED_CORRECTED_MANIFEST_SHA256 = (
    "94f1acd338c933dbc7d45233c8f6a8429e37df1cdc2a9cca356b730a00c573e2"
)


# -----------------------------------------------------------------------------
# Stage26-6D Pareto — must remain immutable
# -----------------------------------------------------------------------------

PARETO_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_6d_cpu_pareto_point_estimate"
)

PARETO_JSON = (
    PARETO_DIR
    / "stage26_6d_point_estimate_frontiers.json"
)

EXPECTED_PARETO_SHA256 = (
    "367b3d34ddb4dd125dcbe3db71291b5576b4521640cd2f11d76e50dc247fc6b5"
)


# -----------------------------------------------------------------------------
# Stage26-5A E2E availability closure
# -----------------------------------------------------------------------------

E2E_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_5a_e2e_availability_closure"
)


# =============================================================================
# 1. EXPECTED BATCH / TARGET UNIVERSE
# =============================================================================

EXPECTED_BATCHES = [
    1,
    64,
    256,
    1024,
    8192,
]


PRIMARY_TARGETS = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
]


OPERATIONAL_TARGETS = [
    "ENS_LGBM_XGB_EQUAL",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
]


ALL_INFERENCE_TARGETS = (
    PRIMARY_TARGETS
    +
    OPERATIONAL_TARGETS
)


GROUP_B_TARGETS = [
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
]


# =============================================================================
# 2. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            p.stdout
        )

    return p


def git(*args):

    return run(
        [
            "git",
            *args,
        ]
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def load_csv(path):

    with Path(path).open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def tracked_files_containing(token):

    rows = git(
        "ls-files"
    ).splitlines()

    token_lower = token.lower()

    return [
        row
        for row in rows
        if token_lower in row.lower()
    ]


def text_file(path):

    return Path(
        path
    ).suffix.lower() in {
        ".json",
        ".csv",
        ".txt",
        ".md",
        ".py",
    }


def summarize_json_structure(path):

    try:

        obj = json.loads(
            Path(path).read_text(
                encoding="utf-8"
            )
        )

    except Exception as exc:

        return {
            "parse":
                "FAIL",

            "error":
                repr(
                    exc
                ),
        }


    if isinstance(
        obj,
        dict,
    ):

        return {
            "parse":
                "PASS",

            "type":
                "dict",

            "top_level_keys":
                list(
                    obj.keys()
                ),
        }


    if isinstance(
        obj,
        list,
    ):

        return {
            "parse":
                "PASS",

            "type":
                "list",

            "length":
                len(
                    obj
                ),
        }


    return {
        "parse":
            "PASS",

        "type":
            type(
                obj
            ).__name__,
    }


def scan_json_for_terms(
    obj,
    terms,
    *,
    path="$",
    out=None,
):

    if out is None:
        out = []


    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            child = (
                path
                +
                "."
                +
                str(
                    key
                )
            )

            key_lower = str(
                key
            ).lower()


            if any(
                term.lower()
                in
                key_lower
                for term in terms
            ):

                out.append(
                    {
                        "path":
                            child,

                        "value":
                            value,
                    }
                )


            scan_json_for_terms(
                value,
                terms,
                path=child,
                out=out,
            )


    elif isinstance(
        obj,
        list,
    ):

        for index, value in enumerate(
            obj
        ):

            scan_json_for_terms(
                value,
                terms,
                path=(
                    path
                    +
                    f"[{index}]"
                ),
                out=out,
            )


    return out


# =============================================================================
# 3. DURABLE GIT GATE
# =============================================================================

banner(
    "STAGE26-7A :: DURABLE SCIENTIFIC STATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)

print(
    "Transient output exists:",
    OUT.exists()
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-7A parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Stage26-7A."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if OUT.exists():

    raise RuntimeError(
        "Stage26-7A transient inventory already exists."
    )


# =============================================================================
# 4. EXACT KNOWN SOURCE IDENTITIES
# =============================================================================

banner(
    "STAGE26-7A :: EXACT KNOWN SOURCE IDENTITIES"
)


identity_checks = [
    (
        "warm raw",
        WARM_RAW,
        EXPECTED_WARM_RAW_SHA256,
    ),

    (
        "warm summary",
        WARM_SUMMARY,
        EXPECTED_WARM_SUMMARY_SHA256,
    ),

    (
        "warm status",
        WARM_STATUS,
        EXPECTED_WARM_STATUS_SHA256,
    ),

    (
        "corrected uncertainty JSON",
        CORRECTED_JSON,
        EXPECTED_CORRECTED_JSON_SHA256,
    ),

    (
        "corrected uncertainty CSV",
        CORRECTED_CSV,
        EXPECTED_CORRECTED_CSV_SHA256,
    ),

    (
        "corrected uncertainty receipt",
        CORRECTED_RECEIPT,
        EXPECTED_CORRECTED_RECEIPT_SHA256,
    ),

    (
        "corrected uncertainty manifest",
        CORRECTED_MANIFEST,
        EXPECTED_CORRECTED_MANIFEST_SHA256,
    ),

    (
        "Stage26-6D Pareto",
        PARETO_JSON,
        EXPECTED_PARETO_SHA256,
    ),
]


known_identity = {}


for label, path, expected in identity_checks:

    if not path.is_file():

        raise FileNotFoundError(
            path
        )


    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:34s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Identity mismatch: {label}"
        )


    known_identity[
        label
    ] = {
        "repo_relative_path":
            str(
                path.relative_to(
                    REPO
                )
            ),

        "sha256":
            actual,
    }


# =============================================================================
# 5. WARM INFERENCE CONDITION GEOMETRY
# =============================================================================

banner(
    "STAGE26-7A :: WARM INFERENCE GEOMETRY"
)


warm_summary = load_csv(
    WARM_SUMMARY
)

warm_status = load_csv(
    WARM_STATUS
)


if len(
    warm_status
) != 80:

    raise RuntimeError(
        f"Expected 80 warm conditions; found {len(warm_status)}."
    )


target_set = sorted(
    set(
        row[
            "target_id"
        ]
        for row in warm_status
    )
)


print(
    "Targets:",
    target_set
)


if set(
    target_set
) != set(
    ALL_INFERENCE_TARGETS
):

    raise RuntimeError(
        "Warm inference target universe mismatch."
    )


warm_geometry = {}


for target in ALL_INFERENCE_TARGETS:

    warm_geometry[
        target
    ] = {}


    for hardware_mode in [
        "CPU_1_PHYSICAL_CORE",
        "CPU_2_PHYSICAL_CORES",
    ]:

        rows = [
            row
            for row in warm_status
            if (
                row[
                    "target_id"
                ]
                ==
                target
                and
                row[
                    "hardware_mode"
                ]
                ==
                hardware_mode
            )
        ]


        batches = sorted(
            int(
                row[
                    "batch_size"
                ]
            )
            for row in rows
        )


        if batches != EXPECTED_BATCHES:

            raise RuntimeError(
                f"{target}/{hardware_mode}: batch universe mismatch."
            )


        statuses = {
            int(
                row[
                    "batch_size"
                ]
            ):
                row[
                    "status"
                ]
            for row in rows
        }


        warm_geometry[
            target
        ][
            hardware_mode
        ] = statuses


    print(
        f"\n{target}"
    )

    print(
        "  CPU1:",
        warm_geometry[
            target
        ][
            "CPU_1_PHYSICAL_CORE"
        ]
    )

    print(
        "  CPU2:",
        warm_geometry[
            target
        ][
            "CPU_2_PHYSICAL_CORES"
        ]
    )


# =============================================================================
# 6. CORRECTED UNCERTAINTY COVERAGE
# =============================================================================

banner(
    "STAGE26-7A :: CORRECTED UNCERTAINTY COVERAGE"
)


corrected = json.loads(
    CORRECTED_JSON.read_text(
        encoding="utf-8"
    )
)


corrected_conditions = corrected[
    "conditions"
]


if len(
    corrected_conditions
) != 80:

    raise RuntimeError(
        "Corrected uncertainty condition count is not 80."
    )


corrected_by_condition = {
    row[
        "condition_id"
    ]:
        row
    for row in corrected_conditions
}


corrected_pass_count = sum(
    1
    for row in corrected_conditions
    if (
        row[
            "status"
        ]
        ==
        "PASS"
        and
        row[
            "bootstrap_computed"
        ]
        is True
    )
)


corrected_nonpass_bootstrap = sum(
    1
    for row in corrected_conditions
    if (
        row[
            "status"
        ]
        !=
        "PASS"
        and
        row[
            "bootstrap_computed"
        ]
        is True
    )
)


print(
    "Corrected PASS intervals:",
    corrected_pass_count
)

print(
    "Non-PASS intervals fabricated:",
    corrected_nonpass_bootstrap
)


if corrected_pass_count != 73:

    raise RuntimeError(
        "Expected corrected uncertainty for 73 PASS conditions."
    )


if corrected_nonpass_bootstrap != 0:

    raise RuntimeError(
        "Corrected uncertainty exists for a non-PASS condition."
    )


# =============================================================================
# 7. DISCOVER EXACT 4B3 / 4C3 / 5A TRACKED ARTIFACTS
# =============================================================================

banner(
    "STAGE26-7A :: DISCOVER COMPONENT SOURCE ARTIFACTS"
)


search_tokens = {
    "raw_extraction_4b3":
        "stage26_4b3",

    "representation_4c3":
        "stage26_4c3",

    "e2e_closure_5a":
        "stage26_5a",
}


discovered = {}


for label, token in search_tokens.items():

    files = tracked_files_containing(
        token
    )


    discovered[
        label
    ] = []


    print(
        "\n",
        label,
        sep="",
    )


    if not files:

        print(
            "  [NO TRACKED FILES FOUND]"
        )


    for relpath in files:

        path = (
            REPO
            /
            relpath
        )


        if not path.is_file():

            continue


        record = {
            "repo_relative_path":
                relpath,

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }


        if path.suffix.lower() == ".json":

            record[
                "json_structure"
            ] = summarize_json_structure(
                path
            )


        discovered[
            label
        ].append(
            record
        )


        print(
            f"  {record['sha256']} "
            f"{record['size_bytes']:10,d} B "
            f"{relpath}"
        )


if not discovered[
    "raw_extraction_4b3"
]:

    raise RuntimeError(
        "No tracked Stage26-4B3 artifacts discovered."
    )


if not discovered[
    "representation_4c3"
]:

    raise RuntimeError(
        "No tracked Stage26-4C3 artifacts discovered."
    )


if not discovered[
    "e2e_closure_5a"
]:

    raise RuntimeError(
        "No tracked Stage26-5A artifacts discovered."
    )


# =============================================================================
# 8. IDENTIFY 4C3 SUMMARY SOURCES
# =============================================================================

banner(
    "STAGE26-7A :: REPRESENTATION SUMMARY SOURCE RESOLUTION"
)


representation_candidates = []


for record in discovered[
    "representation_4c3"
]:

    relpath = record[
        "repo_relative_path"
    ]

    lower = relpath.lower()


    if (
        "summary"
        in
        lower
        and
        Path(
            relpath
        ).suffix.lower()
        in {
            ".json",
            ".csv",
        }
    ):

        representation_candidates.append(
            record
        )


print(
    "Representation summary candidates:",
    len(
        representation_candidates
    )
)


for record in representation_candidates:

    print(
        " ",
        record[
            "sha256"
        ],
        record[
            "repo_relative_path"
        ],
    )


# Known frozen Stage26-4C3 summary hashes.
EXPECTED_REPRESENTATION_SUMMARY_JSON_SHA = (
    "63c80411"
)

EXPECTED_REPRESENTATION_SUMMARY_CSV_SHA = (
    "ca5bbc17"
)


representation_json_matches = [
    record
    for record in representation_candidates
    if (
        record[
            "sha256"
        ].startswith(
            EXPECTED_REPRESENTATION_SUMMARY_JSON_SHA
        )
        and
        Path(
            record[
                "repo_relative_path"
            ]
        ).suffix.lower()
        ==
        ".json"
    )
]


representation_csv_matches = [
    record
    for record in representation_candidates
    if (
        record[
            "sha256"
        ].startswith(
            EXPECTED_REPRESENTATION_SUMMARY_CSV_SHA
        )
        and
        Path(
            record[
                "repo_relative_path"
            ]
        ).suffix.lower()
        ==
        ".csv"
    )
]


if len(
    representation_json_matches
) != 1:

    raise RuntimeError(
        "Could not uniquely resolve frozen Stage26-4C3 summary JSON."
    )


if len(
    representation_csv_matches
) != 1:

    raise RuntimeError(
        "Could not uniquely resolve frozen Stage26-4C3 summary CSV."
    )


representation_json = (
    REPO
    /
    representation_json_matches[
        0
    ][
        "repo_relative_path"
    ]
)

representation_csv = (
    REPO
    /
    representation_csv_matches[
        0
    ][
        "repo_relative_path"
    ]
)


print(
    "\nResolved JSON:"
)

print(
    " ",
    representation_json.relative_to(
        REPO
    )
)

print(
    " ",
    sha256_file(
        representation_json
    )
)


print(
    "\nResolved CSV:"
)

print(
    " ",
    representation_csv.relative_to(
        REPO
    )
)

print(
    " ",
    sha256_file(
        representation_csv
    )
)


# =============================================================================
# 9. REPRESENTATION BATCH / UNIT AUDIT
# =============================================================================

banner(
    "STAGE26-7A :: REPRESENTATION BATCH / UNIT AUDIT"
)


representation_rows = load_csv(
    representation_csv
)


if len(
    representation_rows
) != 5:

    raise RuntimeError(
        f"Expected five representation summary rows; "
        f"found {len(representation_rows)}."
    )


print(
    "Columns:"
)

for column in representation_rows[
    0
].keys():

    print(
        " ",
        column
    )


rep_batches = sorted(
    int(
        row[
            "batch_size"
        ]
    )
    for row in representation_rows
)


if rep_batches != EXPECTED_BATCHES:

    raise RuntimeError(
        "Representation batch universe mismatch."
    )


representation_inventory = []


for row in representation_rows:

    batch = int(
        row[
            "batch_size"
        ]
    )


    # Resolve throughput field conservatively from actual schema.
    throughput_candidates = [
        key
        for key in row.keys()
        if (
            "throughput"
            in key.lower()
            or
            "flows_per_second"
            in key.lower()
        )
    ]


    representation_inventory.append(
        {
            "batch_size":
                batch,

            "throughput_candidate_fields":
                throughput_candidates,

            "row":
                row,
        }
    )


    print(
        f"\nBatch {batch}"
    )

    print(
        "  throughput candidate fields:",
        throughput_candidates
    )


if any(
    len(
        item[
            "throughput_candidate_fields"
        ]
    )
    ==
    0
    for item in representation_inventory
):

    raise RuntimeError(
        "Representation summary has no identifiable throughput field."
    )


# =============================================================================
# 10. RESOLVE RAW EXTRACTION SUMMARY/RECEIPT EVIDENCE
# =============================================================================

banner(
    "STAGE26-7A :: RAW EXTRACTION COMPONENT SOURCE RESOLUTION"
)


raw_extraction_text_evidence = []


for record in discovered[
    "raw_extraction_4b3"
]:

    path = (
        REPO
        /
        record[
            "repo_relative_path"
        ]
    )


    if path.suffix.lower() != ".json":

        continue


    try:

        obj = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )

    except Exception:

        continue


    evidence = scan_json_for_terms(
        obj,
        [
            "packets_per_second",
            "flows_per_second",
            "captured_mib",
            "elapsed",
            "throughput",
            "exportable",
            "lifecycle",
        ],
    )


    if evidence:

        raw_extraction_text_evidence.append(
            {
                "repo_relative_path":
                    record[
                        "repo_relative_path"
                    ],

                "sha256":
                    record[
                        "sha256"
                    ],

                "evidence":
                    evidence,
            }
        )


for artifact in raw_extraction_text_evidence:

    print(
        "\n",
        artifact[
            "repo_relative_path"
        ],
        sep="",
    )

    print(
        "  SHA256:",
        artifact[
            "sha256"
        ]
    )


    for item in artifact[
        "evidence"
    ][
        :30
    ]:

        print(
            " ",
            item[
                "path"
            ],
            "=",
            repr(
                item[
                    "value"
                ]
            ),
        )


if not raw_extraction_text_evidence:

    raise RuntimeError(
        "Could not locate structured Stage26-4B3 throughput evidence."
    )


# =============================================================================
# 11. STAGE26-5A MISSING-BOUNDARY EVIDENCE
# =============================================================================

banner(
    "STAGE26-7A :: STAGE26-5A COMPOSABILITY GATE"
)


e2e_evidence = []


for record in discovered[
    "e2e_closure_5a"
]:

    path = (
        REPO
        /
        record[
            "repo_relative_path"
        ]
    )


    if path.suffix.lower() != ".json":

        continue


    try:

        obj = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )

    except Exception:

        continue


    evidence = scan_json_for_terms(
        obj,
        [
            "complete_e2e",
            "additive",
            "prohibited",
            "available",
            "missing",
            "70_feature",
            "bridge",
            "imputed",
            "status",
        ],
    )


    if evidence:

        e2e_evidence.append(
            {
                "repo_relative_path":
                    record[
                        "repo_relative_path"
                    ],

                "sha256":
                    record[
                        "sha256"
                    ],

                "evidence":
                    evidence,
            }
        )


for artifact in e2e_evidence:

    print(
        "\n",
        artifact[
            "repo_relative_path"
        ],
        sep="",
    )


    for item in artifact[
        "evidence"
    ][
        :40
    ]:

        print(
            " ",
            item[
                "path"
            ],
            "=",
            repr(
                item[
                    "value"
                ]
            ),
        )


if not e2e_evidence:

    raise RuntimeError(
        "Could not recover Stage26-5A missing-boundary evidence."
    )


# =============================================================================
# 12. PROSPECTIVE COMPATIBILITY MATRIX
# =============================================================================

banner(
    "STAGE26-7A :: PROSPECTIVE COMPATIBILITY MATRIX"
)


compatibility = [
    {
        "analysis_id":
            "INFERENCE_BATCH_SCALING_CPU1",

        "status":
            "ALLOWED",

        "scope":
            "ALL_8_STAGE26_INFERENCE_TARGETS",

        "hardware":
            "CPU_1_PHYSICAL_CORE",

        "matching_requirement":
            "WITHIN_TARGET_ACROSS_FROZEN_BATCH_SIZES",

        "point_source":
            str(
                WARM_SUMMARY.relative_to(
                    REPO
                )
            ),

        "uncertainty_source":
            str(
                CORRECTED_JSON.relative_to(
                    REPO
                )
            ),

        "claim":
            "INFERENCE_COMPONENT_BATCH_SCALING_ONLY",
    },

    {
        "analysis_id":
            "INFERENCE_BATCH_SCALING_CPU2",

        "status":
            "ALLOWED",

        "scope":
            "ALL_8_STAGE26_INFERENCE_TARGETS",

        "hardware":
            "CPU_2_PHYSICAL_CORES",

        "matching_requirement":
            "WITHIN_TARGET_ACROSS_FROZEN_BATCH_SIZES",

        "point_source":
            str(
                WARM_SUMMARY.relative_to(
                    REPO
                )
            ),

        "uncertainty_source":
            str(
                CORRECTED_JSON.relative_to(
                    REPO
                )
            ),

        "claim":
            "INFERENCE_COMPONENT_BATCH_SCALING_ONLY",
    },

    {
        "analysis_id":
            "GROUP_B_REPRESENTATION_VS_INFERENCE_CPU1",

        "status":
            "ALLOWED_COMPONENT_LEVEL_ONLY",

        "scope":
            GROUP_B_TARGETS,

        "hardware":
            "CPU_1_PHYSICAL_CORE",

        "matching_requirement":
            (
                "EXACT_BATCH_MATCH_AND_BOTH_COMPONENTS_PASS"
            ),

        "representation_source":
            str(
                representation_csv.relative_to(
                    REPO
                )
            ),

        "inference_source":
            str(
                WARM_SUMMARY.relative_to(
                    REPO
                )
            ),

        "ratio_definition":
            (
                "InferenceThroughput / RepresentationThroughput"
            ),

        "ratio_interpretation":
            (
                ">1 representation component has lower capacity; "
                "<1 inference component has lower capacity."
            ),

        "claim":
            (
                "DOWNSTREAM_COMPONENT_CAPACITY_RATIO_ONLY__"
                "NOT_COMPLETE_E2E"
            ),
    },

    {
        "analysis_id":
            "RAW_EXTRACTION_COMPONENT_REPORTING",

        "status":
            "ALLOWED_DESCRIPTIVE_ONLY",

        "scope":
            "STAGE26_4B3_CPU1_RAW_PCAP_TO_FLOW_RECONSTRUCTION",

        "claim":
            (
                "RAW_EXTRACTION_COMPONENT_THROUGHPUT_ONLY__"
                "NO_MODEL_PIPELINE_BOTTLENECK_INFERENCE"
            ),
    },

    {
        "analysis_id":
            "GROUP_A_EXTRACTION_VS_INFERENCE",

        "status":
            "PROHIBITED",

        "reason":
            (
                "FROZEN_70_FEATURE_EXTRACTION_PATH_NOT_PROFILED"
            ),
    },

    {
        "analysis_id":
            "RAW_EXTRACTION_TO_GROUP_B_INFERENCE",

        "status":
            "PROHIBITED_AS_PIPELINE_RATIO",

        "reason":
            (
                "RAW_FLOW_TO_COMPACT_PACKET_IMAGE_CORPUS_BRIDGE_NOT_MEASURED"
            ),
    },

    {
        "analysis_id":
            "RAW_EXTRACTION_PLUS_REPRESENTATION_PLUS_INFERENCE_ADDITIVE_E2E",

        "status":
            "PROHIBITED",

        "reason":
            (
                "STAGE26_5A_CLOSED_NO_VALID_COMPLETE_E2E_MEASUREMENT"
            ),
    },

    {
        "analysis_id":
            "CROSS_GROUP_PARETO_FROM_CAPACITY",

        "status":
            "PROHIBITED",

        "reason":
            "NO_CROSS_GROUP_PARETO_FRONTIER",
    },
]


for item in compatibility:

    print(
        f"{item['analysis_id']:58s} "
        f"{item['status']}"
    )


# =============================================================================
# 13. MATCHED GROUP-B CPU1 BATCH AVAILABILITY
# =============================================================================

banner(
    "STAGE26-7A :: GROUP-B MATCHED CPU1 BATCH AVAILABILITY"
)


representation_batches = set(
    rep_batches
)


group_b_matched_batches = {}


for target in GROUP_B_TARGETS:

    cpu1_status = warm_geometry[
        target
    ][
        "CPU_1_PHYSICAL_CORE"
    ]


    matched = []


    for batch in EXPECTED_BATCHES:

        inference_status = cpu1_status[
            batch
        ]


        representation_available = (
            batch
            in
            representation_batches
        )


        eligible = (
            representation_available
            and
            inference_status
            ==
            "PASS"
        )


        matched.append(
            {
                "batch_size":
                    batch,

                "representation_available":
                    representation_available,

                "inference_status":
                    inference_status,

                "component_ratio_eligible":
                    eligible,
            }
        )


    group_b_matched_batches[
        target
    ] = matched


    print(
        "\n",
        target,
        sep="",
    )


    for row in matched:

        print(
            f"  B={row['batch_size']:5d} "
            f"representation={'YES' if row['representation_available'] else 'NO ':3s} "
            f"inference={row['inference_status']:24s} "
            f"ratio={'YES' if row['component_ratio_eligible'] else 'NO'}"
        )


# =============================================================================
# 14. WRITE TRANSIENT INVENTORY
# =============================================================================

banner(
    "STAGE26-7A :: WRITE TRANSIENT INVENTORY"
)


CAPACITY_RUNTIME.mkdir(
    parents=True,
    exist_ok=True,
)


payload = {
    "schema":
        "stage26_7a_capacity_source_compatibility_inventory_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "status":
        "PASS_SOURCE_AND_COMPATIBILITY_RESOLUTION_BEFORE_CAPACITY_CALCULATION",

    "known_source_identity":
        known_identity,

    "discovered_component_artifacts":
        discovered,

    "resolved_representation_sources": {
        "summary_json": {
            "repo_relative_path":
                str(
                    representation_json.relative_to(
                        REPO
                    )
                ),

            "sha256":
                sha256_file(
                    representation_json
                ),
        },

        "summary_csv": {
            "repo_relative_path":
                str(
                    representation_csv.relative_to(
                        REPO
                    )
                ),

            "sha256":
                sha256_file(
                    representation_csv
                ),
        },
    },

    "representation_inventory":
        representation_inventory,

    "raw_extraction_structured_evidence":
        raw_extraction_text_evidence,

    "stage26_5a_composability_evidence":
        e2e_evidence,

    "inference_geometry":
        warm_geometry,

    "group_b_matched_CPU1_batches":
        group_b_matched_batches,

    "compatibility_matrix":
        compatibility,

    "prospective_stage26_7_rules": {
        "batch_scaling_may_be_computed":
            True,

        "CPU1_and_CPU2_must_be_analyzed_separately":
            True,

        "OOM_timeout_preserved_as_resource_limits":
            True,

        "Stage26_6F1_corrected_uncertainty_is_publication_uncertainty_source":
            True,

        "historical_Stage26_2_CIs_are_publication_uncertainty_source":
            False,

        "Group_B_representation_inference_ratio_allowed":
            True,

        "Group_B_ratio_CPU1_only":
            True,

        "Group_B_ratio_requires_exact_batch_match":
            True,

        "Group_B_ratio_is_complete_E2E":
            False,

        "Group_A_extraction_inference_ratio_allowed":
            False,

        "raw_extraction_to_Group_B_model_pipeline_ratio_allowed":
            False,

        "additive_A_plus_B_plus_C_complete_E2E_allowed":
            False,

        "missing_cost_imputation_allowed":
            False,

        "Pareto_membership_may_be_recomputed":
            False,

        "cross_group_Pareto_allowed":
            False,

        "GPU_allowed":
            False,
    },

    "scientific_state": {
        "capacity_ratio_computed":
            False,

        "batch_speedup_computed":
            False,

        "scaling_efficiency_computed":
            False,

        "bottleneck_declared":
            False,

        "Pareto_recomputed":
            False,

        "bootstrap_executed":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "PCAP_accessed":
            False,

        "Release_corpus_accessed":
            False,

        "GPU_used":
            False,

        "Git_modified":
            False,
    },

    "next":
        (
            "After review, freeze these exact Stage26-7 source identities "
            "and compatibility rules in Git before computing batch-scaling "
            "or component-capacity ratios."
        ),
}


with OUT.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        payload,
        f,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    )

    f.write(
        "\n"
    )


out_sha = sha256_file(
    OUT
)


print(
    "Output:"
)

print(
    " ",
    OUT
)

print(
    "SHA256:"
)

print(
    " ",
    out_sha
)


# =============================================================================
# 15. FINAL READ-ONLY CLOSURE
# =============================================================================

banner(
    "STAGE26-7A SOURCE / COMPATIBILITY INVENTORY COMPLETE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed during Stage26-7A."
    )


if final_remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed during Stage26-7A."
    )


if final_status:

    raise RuntimeError(
        "Stage26-7A unexpectedly modified Git."
    )


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  exact inference sources resolved       : YES"
)

print(
    "  corrected uncertainty source resolved  : YES"
)

print(
    "  representation sources resolved        : YES"
)

print(
    "  raw extraction sources inventoried     : YES"
)

print(
    "  Stage26-5A composability audited       : YES"
)

print(
    "  compatible analyses classified         : YES"
)

print(
    "  capacity ratio computed                : NO"
)

print(
    "  batch scaling computed                 : NO"
)

print(
    "  bottleneck declared                    : NO"
)

print(
    "  complete E2E reconstructed             : NO"
)

print(
    "  missing cost imputed                   : NO"
)

print(
    "  Pareto changed                         : NO"
)

print(
    "  new inference                          : NO"
)

print(
    "  new timing                             : NO"
)

print(
    "  GPU                                    : NO"
)

print(
    "  Git modified                           : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Review exact representation throughput field(s),"
)

print(
    "  raw-extraction evidence, and matched Group-B CPU1 batches."
)

print(
    "  Then Git-freeze the Stage26-7 compatibility/source lock"
)

print(
    "  BEFORE computing any scaling or component-capacity ratio."
)


STAGE26-7A :: DURABLE SCIENTIFIC STATE
Expected parent: a484148cd8ea7d7c89604d3a015f73bd6f1bc81a
Local HEAD     : a484148cd8ea7d7c89604d3a015f73bd6f1bc81a
origin/main    : a484148cd8ea7d7c89604d3a015f73bd6f1bc81a
Repo clean     : True
Transient output exists: False

STAGE26-7A :: EXACT KNOWN SOURCE IDENTITIES
warm raw                           PASS 78c58289ccfc4598966d6516201028f43c4b1d16d8bd0268a0cf58129d4179fa
warm summary                       PASS 75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232
warm status                        PASS df146563826992cb57702e589e4cf5d875ecfdbbf79533a731405e8eb738e7af
corrected uncertainty JSON         PASS 45ea4408738fb5101ce68a363c2cc0a7951648eed49e474da9489b00ce6128ce
corrected uncertainty CSV          PASS f804f117312b967bfc13cc070a34be43545238bfd30a758a00ad58202c9e6828
corrected uncertainty receipt      PASS 8d37b6bc2e0a7a75b8102b7d1cd3c1c058907420b229c9da1f302d97a44f1130
corrected uncertainty manifest     PASS 94f1acd338c933dbc7

RuntimeError: STAGE16_XGBOOST_TUNED/CPU_2_PHYSICAL_CORES: batch universe mismatch.

In [24]:
# =============================================================================
# STAGE26-7A-DIAG
# NARROW DIAGNOSIS OF CPU2 HARDWARE-MODE / BATCH GEOMETRY
#
# PURPOSE
# -------
# Determine why Cell 40 did not resolve the expected five XGBoost CPU2
# conditions.
#
# READ ONLY.
#
# NO:
#   - scaling calculation
#   - throughput ratio
#   - bottleneck declaration
#   - bootstrap
#   - inference
#   - timing
#   - Pareto recomputation
#   - Git write
#   - GPU
# =============================================================================

from __future__ import annotations

import csv
import hashlib
import subprocess
from collections import Counter, defaultdict
from pathlib import Path


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "a484148cd8ea7d7c89604d3a015f73bd6f1bc81a"
)

WARM_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_2_cpu_warm_inference"
)

WARM_STATUS = (
    WARM_DIR
    / "stage26_2_condition_status.csv"
)

WARM_SUMMARY = (
    WARM_DIR
    / "stage26_2_warm_summary.csv"
)

EXPECTED_WARM_STATUS_SHA256 = (
    "df146563826992cb57702e589e4cf5d875ecfdbbf79533a731405e8eb738e7af"
)

EXPECTED_WARM_SUMMARY_SHA256 = (
    "75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232"
)

TARGET = (
    "STAGE16_XGBOOST_TUNED"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def git(*args):

    p = subprocess.run(
        ["git", *args],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def load_csv(path):

    with Path(path).open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


# =============================================================================
# 2. DURABLE STATE
# =============================================================================

banner(
    "STAGE26-7A-DIAG :: DURABLE STATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed after failed Stage26-7A."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed after failed Stage26-7A."
    )


if status:

    raise RuntimeError(
        "Repository is not clean."
    )


# =============================================================================
# 3. SOURCE IDENTITY
# =============================================================================

banner(
    "STAGE26-7A-DIAG :: SOURCE IDENTITY"
)


status_sha = sha256_file(
    WARM_STATUS
)

summary_sha = sha256_file(
    WARM_SUMMARY
)


print(
    "Status expected:",
    EXPECTED_WARM_STATUS_SHA256
)

print(
    "Status actual  :",
    status_sha
)

print(
    "Summary expected:",
    EXPECTED_WARM_SUMMARY_SHA256
)

print(
    "Summary actual  :",
    summary_sha
)


if status_sha != EXPECTED_WARM_STATUS_SHA256:

    raise RuntimeError(
        "Stage26-2 status identity changed."
    )


if summary_sha != EXPECTED_WARM_SUMMARY_SHA256:

    raise RuntimeError(
        "Stage26-2 summary identity changed."
    )


# =============================================================================
# 4. LOAD TABLES
# =============================================================================

status_rows = load_csv(
    WARM_STATUS
)

summary_rows = load_csv(
    WARM_SUMMARY
)


print(
    "\nStatus rows :",
    len(
        status_rows
    )
)

print(
    "Summary rows:",
    len(
        summary_rows
    )
)


# =============================================================================
# 5. GLOBAL LITERAL HARDWARE-MODE INVENTORY
# =============================================================================

banner(
    "STAGE26-7A-DIAG :: GLOBAL HARDWARE-MODE LITERALS"
)


hardware_counter = Counter(
    row[
        "hardware_mode"
    ]
    for row in status_rows
)


for value, count in sorted(
    hardware_counter.items(),
    key=lambda item: repr(
        item[
            0
        ]
    ),
):

    print(
        f"{repr(value):40s} count={count}"
    )


print(
    "\nHardware-mode strings with byte representations:"
)


for value in sorted(
    hardware_counter
):

    print(
        " ",
        repr(
            value
        ),
        "->",
        value.encode(
            "utf-8"
        ).hex(),
    )


# =============================================================================
# 6. XGBOOST — EVERY STATUS ROW, UNFILTERED
# =============================================================================

banner(
    "STAGE26-7A-DIAG :: XGBOOST STATUS ROWS — UNFILTERED"
)


xgb_status = [
    row
    for row in status_rows
    if row[
        "target_id"
    ]
    ==
    TARGET
]


print(
    "XGBoost status-row count:",
    len(
        xgb_status
    )
)


for row in xgb_status:

    print(
        "  condition_id=",
        repr(
            row[
                "condition_id"
            ]
        ),
        " hardware_mode=",
        repr(
            row[
                "hardware_mode"
            ]
        ),
        " thread_count=",
        repr(
            row[
                "thread_count"
            ]
        ),
        " batch_size=",
        repr(
            row[
                "batch_size"
            ]
        ),
        " status=",
        repr(
            row[
                "status"
            ]
        ),
        sep="",
    )


# =============================================================================
# 7. XGBOOST GROUPED BY LITERAL HARDWARE MODE
# =============================================================================

banner(
    "STAGE26-7A-DIAG :: XGBOOST MODE/BATCH GEOMETRY"
)


xgb_by_mode = defaultdict(
    list
)


for row in xgb_status:

    xgb_by_mode[
        row[
            "hardware_mode"
        ]
    ].append(
        row
    )


for mode in sorted(
    xgb_by_mode
):

    rows = xgb_by_mode[
        mode
    ]

    batches = [
        int(
            row[
                "batch_size"
            ]
        )
        for row in rows
    ]


    print(
        "\nMode:",
        repr(
            mode
        )
    )

    print(
        "  row count:",
        len(
            rows
        )
    )

    print(
        "  batches  :",
        sorted(
            batches
        )
    )

    print(
        "  threads  :",
        sorted(
            set(
                int(
                    row[
                        "thread_count"
                    ]
                )
                for row in rows
            )
        )
    )

    print(
        "  statuses :",
        {
            int(
                row[
                    "batch_size"
                ]
            ):
                row[
                    "status"
                ]
            for row in rows
        }
    )


# =============================================================================
# 8. GLOBAL TARGET × HARDWARE-MODE GEOMETRY
# =============================================================================

banner(
    "STAGE26-7A-DIAG :: GLOBAL TARGET × HARDWARE-MODE GEOMETRY"
)


targets = sorted(
    set(
        row[
            "target_id"
        ]
        for row in status_rows
    )
)


global_geometry = {}


for target in targets:

    rows_target = [
        row
        for row in status_rows
        if row[
            "target_id"
        ]
        ==
        target
    ]


    by_mode = defaultdict(
        list
    )


    for row in rows_target:

        by_mode[
            row[
                "hardware_mode"
            ]
        ].append(
            row
        )


    global_geometry[
        target
    ] = {}


    print(
        "\n",
        target,
        sep="",
    )


    for mode in sorted(
        by_mode
    ):

        rows = by_mode[
            mode
        ]

        batches = sorted(
            int(
                row[
                    "batch_size"
                ]
            )
            for row in rows
        )


        global_geometry[
            target
        ][
            mode
        ] = batches


        print(
            "  ",
            repr(
                mode
            ),
            " -> ",
            batches,
            sep="",
        )


# =============================================================================
# 9. CROSS-CHECK SUMMARY USES SAME MODE GEOMETRY
# =============================================================================

banner(
    "STAGE26-7A-DIAG :: STATUS / SUMMARY MODE CROSS-CHECK"
)


status_keys = sorted(
    (
        row[
            "condition_id"
        ],
        row[
            "target_id"
        ],
        row[
            "hardware_mode"
        ],
        row[
            "thread_count"
        ],
        row[
            "batch_size"
        ],
    )
    for row in status_rows
    if row[
        "status"
    ]
    ==
    "PASS"
)


summary_keys = sorted(
    (
        row[
            "condition_id"
        ],
        row[
            "target_id"
        ],
        row[
            "hardware_mode"
        ],
        row[
            "thread_count"
        ],
        row[
            "batch_size"
        ],
    )
    for row in summary_rows
)


print(
    "PASS status key count:",
    len(
        status_keys
    )
)

print(
    "Summary key count    :",
    len(
        summary_keys
    )
)

print(
    "Exact key equality   :",
    status_keys
    ==
    summary_keys
)


if status_keys != summary_keys:

    only_status = sorted(
        set(
            status_keys
        )
        -
        set(
            summary_keys
        )
    )

    only_summary = sorted(
        set(
            summary_keys
        )
        -
        set(
            status_keys
        )
    )


    print(
        "\nOnly in PASS status:"
    )

    for item in only_status[
        :30
    ]:

        print(
            " ",
            item
        )


    print(
        "\nOnly in summary:"
    )

    for item in only_summary[
        :30
    ]:

        print(
            " ",
            item
        )


# =============================================================================
# 10. DIAGNOSIS
# =============================================================================

banner(
    "STAGE26-7A-DIAG :: DIAGNOSIS"
)


canonical_modes = sorted(
    hardware_counter
)


print(
    "Literal hardware-mode universe:"
)

for mode in canonical_modes:

    print(
        " ",
        repr(
            mode
        )
    )


five_batch_modes = []


for mode in canonical_modes:

    all_targets_have_five = all(
        sorted(
            global_geometry[
                target
            ].get(
                mode,
                []
            )
        )
        ==
        [
            1,
            64,
            256,
            1024,
            8192,
        ]
        for target in targets
    )


    print(
        f"\n{repr(mode)}"
    )

    print(
        "  all 8 targets have exact five-batch geometry:",
        all_targets_have_five
    )


    if all_targets_have_five:

        five_batch_modes.append(
            mode
        )


print(
    "\nModes with exact five-batch geometry for all targets:"
)

print(
    " ",
    five_batch_modes
)


# =============================================================================
# 11. FINAL READ-ONLY CLOSURE
# =============================================================================

banner(
    "STAGE26-7A-DIAG COMPLETE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed during diagnosis."
    )


if final_remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed during diagnosis."
    )


if final_status:

    raise RuntimeError(
        "Diagnostic unexpectedly modified Git."
    )


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  hardware-mode literals inspected : YES"
)

print(
    "  target/batch geometry inspected   : YES"
)

print(
    "  status/summary cross-check        : YES"
)

print(
    "  scaling calculated                : NO"
)

print(
    "  capacity ratio calculated         : NO"
)

print(
    "  bottleneck declared               : NO"
)

print(
    "  Pareto changed                    : NO"
)

print(
    "  new inference                     : NO"
)

print(
    "  new timing                        : NO"
)

print(
    "  GPU                               : NO"
)

print(
    "  Git modified                      : NO"
)


STAGE26-7A-DIAG :: DURABLE STATE
Expected parent: a484148cd8ea7d7c89604d3a015f73bd6f1bc81a
Local HEAD     : a484148cd8ea7d7c89604d3a015f73bd6f1bc81a
origin/main    : a484148cd8ea7d7c89604d3a015f73bd6f1bc81a
Repo clean     : True

STAGE26-7A-DIAG :: SOURCE IDENTITY
Status expected: df146563826992cb57702e589e4cf5d875ecfdbbf79533a731405e8eb738e7af
Status actual  : df146563826992cb57702e589e4cf5d875ecfdbbf79533a731405e8eb738e7af
Summary expected: 75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232
Summary actual  : 75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232

Status rows : 80
Summary rows: 73

STAGE26-7A-DIAG :: GLOBAL HARDWARE-MODE LITERALS
'CPU_1_PHYSICAL_CORE'                    count=40
'CPU_2_PHYSICAL_CORE'                    count=40

Hardware-mode strings with byte representations:
  'CPU_1_PHYSICAL_CORE' -> 4350555f315f504859534943414c5f434f5245
  'CPU_2_PHYSICAL_CORE' -> 4350555f325f504859534943414c5f434f5245

STAGE26-7A-DIAG :: XGBOOST STATUS RO

In [25]:
# =============================================================================
# STAGE26-7A-R
# NARROW RECOVERY OF CPU CAPACITY / COMPATIBILITY INVENTORY
#
# ONLY CORRECTION FROM FAILED CELL 40:
#
#   WRONG:
#       CPU_2_PHYSICAL_CORES
#
#   FROZEN ACTUAL:
#       CPU_2_PHYSICAL_CORE
#
# DIAGNOSTIC CONFIRMED:
#   - 40 CPU1 + 40 CPU2 conditions
#   - all 8 targets have batches [1,64,256,1024,8192] in both modes
#   - PASS-status keys exactly equal Stage26-2 summary keys
#
# THIS CELL:
#   - re-verifies durable state / source hashes
#   - uses exact frozen CPU-mode literals
#   - resolves Stage26-4B3 / 4C3 / 5A sources
#   - determines matched Group-B CPU1 representation/inference batches
#   - freezes NOTHING yet
#   - computes NO ratios/scaling/bottleneck
#   - writes only a transient Stage26-7A inventory outside Git
#
# NO:
#   - throughput ratio calculation
#   - speedup calculation
#   - scaling efficiency
#   - bottleneck declaration
#   - Pareto recomputation
#   - bootstrap
#   - model loading
#   - inference
#   - timing
#   - PCAP access
#   - Release corpus access
#   - GPU
#   - Git modification
# =============================================================================

from __future__ import annotations

import csv
import json
import hashlib
import subprocess
from collections import Counter
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. DURABLE IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "a484148cd8ea7d7c89604d3a015f73bd6f1bc81a"
)

RUNTIME_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

CAPACITY_RUNTIME = (
    RUNTIME_ROOT
    / "capacity"
)

OUT = (
    CAPACITY_RUNTIME
    / "stage26_7a_capacity_source_compatibility_inventory.json"
)


# Exact frozen Stage26 hardware-mode literals.
CPU1 = (
    "CPU_1_PHYSICAL_CORE"
)

CPU2 = (
    "CPU_2_PHYSICAL_CORE"
)


EXPECTED_BATCHES = [
    1,
    64,
    256,
    1024,
    8192,
]


# =============================================================================
# 1. STAGE26-2 SOURCES
# =============================================================================

WARM_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_2_cpu_warm_inference"
)

WARM_RAW = (
    WARM_DIR
    / "stage26_2_warm_raw.csv"
)

WARM_SUMMARY = (
    WARM_DIR
    / "stage26_2_warm_summary.csv"
)

WARM_STATUS = (
    WARM_DIR
    / "stage26_2_condition_status.csv"
)


EXPECTED_WARM_RAW_SHA256 = (
    "78c58289ccfc4598966d6516201028f43c4b1d16d8bd0268a0cf58129d4179fa"
)

EXPECTED_WARM_SUMMARY_SHA256 = (
    "75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232"
)

EXPECTED_WARM_STATUS_SHA256 = (
    "df146563826992cb57702e589e4cf5d875ecfdbbf79533a731405e8eb738e7af"
)


# =============================================================================
# 2. STAGE26-6F1 CORRECTED UNCERTAINTY
# =============================================================================

CORRECTED_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_6f1_bootstrap_corrected_uncertainty"
)

CORRECTED_JSON = (
    CORRECTED_DIR
    / "stage26_6f1_bootstrap_corrected_uncertainty.json"
)

CORRECTED_CSV = (
    CORRECTED_DIR
    / "stage26_6f1_corrected_timing_uncertainty.csv"
)

CORRECTED_RECEIPT = (
    CORRECTED_DIR
    / "stage26_6f1_correction_execution_receipt.json"
)

CORRECTED_MANIFEST = (
    CORRECTED_DIR
    / "stage26_6f1_corrected_uncertainty_manifest.json"
)


EXPECTED_CORRECTED_JSON_SHA256 = (
    "45ea4408738fb5101ce68a363c2cc0a7951648eed49e474da9489b00ce6128ce"
)

EXPECTED_CORRECTED_CSV_SHA256 = (
    "f804f117312b967bfc13cc070a34be43545238bfd30a758a00ad58202c9e6828"
)

EXPECTED_CORRECTED_RECEIPT_SHA256 = (
    "8d37b6bc2e0a7a75b8102b7d1cd3c1c058907420b229c9da1f302d97a44f1130"
)

EXPECTED_CORRECTED_MANIFEST_SHA256 = (
    "94f1acd338c933dbc7d45233c8f6a8429e37df1cdc2a9cca356b730a00c573e2"
)


# =============================================================================
# 3. STAGE26-6D PARETO
# =============================================================================

PARETO_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_6d_cpu_pareto_point_estimate"
)

PARETO_JSON = (
    PARETO_DIR
    / "stage26_6d_point_estimate_frontiers.json"
)

EXPECTED_PARETO_SHA256 = (
    "367b3d34ddb4dd125dcbe3db71291b5576b4521640cd2f11d76e50dc247fc6b5"
)


# =============================================================================
# 4. TARGET UNIVERSE
# =============================================================================

PRIMARY_TARGETS = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
]


OPERATIONAL_TARGETS = [
    "ENS_LGBM_XGB_EQUAL",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
]


ALL_INFERENCE_TARGETS = (
    PRIMARY_TARGETS
    +
    OPERATIONAL_TARGETS
)


GROUP_B_TARGETS = [
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
]


# =============================================================================
# 5. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        +
        "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            p.stdout
        )

    return p


def git(*args):

    return run(
        [
            "git",
            *args,
        ]
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def load_csv(path):

    with Path(path).open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def tracked_files_containing(token):

    token = token.lower()

    return [
        row
        for row in git(
            "ls-files"
        ).splitlines()
        if token in row.lower()
    ]


def scan_json_for_terms(
    obj,
    terms,
    *,
    path="$",
    out=None,
):

    if out is None:

        out = []


    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            child = (
                path
                +
                "."
                +
                str(
                    key
                )
            )

            key_lower = str(
                key
            ).lower()


            if any(
                term.lower()
                in
                key_lower
                for term in terms
            ):

                out.append(
                    {
                        "path":
                            child,

                        "value":
                            value,
                    }
                )


            scan_json_for_terms(
                value,
                terms,
                path=child,
                out=out,
            )


    elif isinstance(
        obj,
        list,
    ):

        for index, value in enumerate(
            obj
        ):

            scan_json_for_terms(
                value,
                terms,
                path=(
                    path
                    +
                    f"[{index}]"
                ),
                out=out,
            )


    return out


# =============================================================================
# 6. DURABLE STATE
# =============================================================================

banner(
    "STAGE26-7A-R :: DURABLE SCIENTIFIC STATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)

print(
    "Transient output exists:",
    OUT.exists()
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-7A-R parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Stage26-7A-R."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if OUT.exists():

    raise RuntimeError(
        "Stage26-7A transient output already exists."
    )


# =============================================================================
# 7. EXACT SOURCE IDENTITIES
# =============================================================================

banner(
    "STAGE26-7A-R :: EXACT SOURCE IDENTITIES"
)


identity_checks = [
    (
        "warm raw",
        WARM_RAW,
        EXPECTED_WARM_RAW_SHA256,
    ),

    (
        "warm summary",
        WARM_SUMMARY,
        EXPECTED_WARM_SUMMARY_SHA256,
    ),

    (
        "warm status",
        WARM_STATUS,
        EXPECTED_WARM_STATUS_SHA256,
    ),

    (
        "corrected uncertainty JSON",
        CORRECTED_JSON,
        EXPECTED_CORRECTED_JSON_SHA256,
    ),

    (
        "corrected uncertainty CSV",
        CORRECTED_CSV,
        EXPECTED_CORRECTED_CSV_SHA256,
    ),

    (
        "corrected uncertainty receipt",
        CORRECTED_RECEIPT,
        EXPECTED_CORRECTED_RECEIPT_SHA256,
    ),

    (
        "corrected uncertainty manifest",
        CORRECTED_MANIFEST,
        EXPECTED_CORRECTED_MANIFEST_SHA256,
    ),

    (
        "Stage26-6D Pareto",
        PARETO_JSON,
        EXPECTED_PARETO_SHA256,
    ),
]


known_identity = {}


for label, path, expected in identity_checks:

    if not path.is_file():

        raise FileNotFoundError(
            path
        )


    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:34s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Identity mismatch: {label}"
        )


    known_identity[
        label
    ] = {
        "repo_relative_path":
            str(
                path.relative_to(
                    REPO
                )
            ),

        "sha256":
            actual,
    }


# =============================================================================
# 8. EXACT HARDWARE-MODE / BATCH GEOMETRY
# =============================================================================

banner(
    "STAGE26-7A-R :: EXACT WARM INFERENCE GEOMETRY"
)


status_rows = load_csv(
    WARM_STATUS
)

summary_rows = load_csv(
    WARM_SUMMARY
)


if len(
    status_rows
) != 80:

    raise RuntimeError(
        f"Expected 80 conditions; found {len(status_rows)}."
    )


hardware_modes = sorted(
    set(
        row[
            "hardware_mode"
        ]
        for row in status_rows
    )
)


print(
    "Frozen hardware-mode universe:"
)

for mode in hardware_modes:

    print(
        " ",
        repr(
            mode
        )
    )


if hardware_modes != [
    CPU1,
    CPU2,
]:

    raise RuntimeError(
        "Unexpected frozen hardware-mode universe."
    )


targets = sorted(
    set(
        row[
            "target_id"
        ]
        for row in status_rows
    )
)


if set(
    targets
) != set(
    ALL_INFERENCE_TARGETS
):

    raise RuntimeError(
        "Inference target universe mismatch."
    )


warm_geometry = {}


for target in ALL_INFERENCE_TARGETS:

    warm_geometry[
        target
    ] = {}


    print(
        "\n",
        target,
        sep="",
    )


    for hardware_mode in [
        CPU1,
        CPU2,
    ]:

        rows = [
            row
            for row in status_rows
            if (
                row[
                    "target_id"
                ]
                ==
                target
                and
                row[
                    "hardware_mode"
                ]
                ==
                hardware_mode
            )
        ]


        batches = sorted(
            int(
                row[
                    "batch_size"
                ]
            )
            for row in rows
        )


        if batches != EXPECTED_BATCHES:

            raise RuntimeError(
                f"{target}/{hardware_mode}: "
                f"batch universe mismatch: {batches}"
            )


        expected_threads = (
            1
            if hardware_mode == CPU1
            else
            2
        )


        actual_threads = sorted(
            set(
                int(
                    row[
                        "thread_count"
                    ]
                )
                for row in rows
            )
        )


        if actual_threads != [
            expected_threads
        ]:

            raise RuntimeError(
                f"{target}/{hardware_mode}: "
                f"thread-count mismatch: {actual_threads}"
            )


        statuses = {
            int(
                row[
                    "batch_size"
                ]
            ):
                row[
                    "status"
                ]
            for row in rows
        }


        warm_geometry[
            target
        ][
            hardware_mode
        ] = statuses


        print(
            f"  {hardware_mode}:",
            statuses
        )


# PASS summary must exactly correspond to PASS condition-status keys.
status_pass_keys = sorted(
    (
        row[
            "condition_id"
        ],
        row[
            "target_id"
        ],
        row[
            "hardware_mode"
        ],
        row[
            "thread_count"
        ],
        row[
            "batch_size"
        ],
    )
    for row in status_rows
    if row[
        "status"
    ]
    ==
    "PASS"
)


summary_keys = sorted(
    (
        row[
            "condition_id"
        ],
        row[
            "target_id"
        ],
        row[
            "hardware_mode"
        ],
        row[
            "thread_count"
        ],
        row[
            "batch_size"
        ],
    )
    for row in summary_rows
)


print(
    "\nPASS status rows:",
    len(
        status_pass_keys
    )
)

print(
    "Summary rows    :",
    len(
        summary_keys
    )
)

print(
    "Exact equality  :",
    status_pass_keys
    ==
    summary_keys
)


if status_pass_keys != summary_keys:

    raise RuntimeError(
        "Stage26-2 PASS status/summary geometry mismatch."
    )


# =============================================================================
# 9. CORRECTED UNCERTAINTY COVERAGE
# =============================================================================

banner(
    "STAGE26-7A-R :: CORRECTED UNCERTAINTY COVERAGE"
)


corrected = json.loads(
    CORRECTED_JSON.read_text(
        encoding="utf-8"
    )
)


corrected_conditions = corrected[
    "conditions"
]


if len(
    corrected_conditions
) != 80:

    raise RuntimeError(
        "Corrected uncertainty condition count is not 80."
    )


corrected_pass = [
    row
    for row in corrected_conditions
    if row[
        "status"
    ]
    ==
    "PASS"
]


corrected_nonpass = [
    row
    for row in corrected_conditions
    if row[
        "status"
    ]
    !=
    "PASS"
]


if len(
    corrected_pass
) != 73:

    raise RuntimeError(
        "Corrected uncertainty PASS count is not 73."
    )


if any(
    row[
        "bootstrap_computed"
    ]
    is not True
    for row in corrected_pass
):

    raise RuntimeError(
        "A PASS condition lacks corrected uncertainty."
    )


if any(
    row[
        "bootstrap_computed"
    ]
    is not False
    for row in corrected_nonpass
):

    raise RuntimeError(
        "A non-PASS condition has fabricated uncertainty."
    )


print(
    "PASS uncertainty available:",
    len(
        corrected_pass
    )
)

print(
    "Non-PASS uncertainty fabricated:",
    0
)


# =============================================================================
# 10. DISCOVER STAGE26-4B3 / 4C3 / 5A TRACKED ARTIFACTS
# =============================================================================

banner(
    "STAGE26-7A-R :: COMPONENT SOURCE DISCOVERY"
)


search_tokens = {
    "raw_extraction_4b3":
        "stage26_4b3",

    "representation_4c3":
        "stage26_4c3",

    "e2e_closure_5a":
        "stage26_5a",
}


discovered = {}


for label, token in search_tokens.items():

    files = tracked_files_containing(
        token
    )


    discovered[
        label
    ] = []


    print(
        "\n",
        label,
        sep="",
    )


    for relpath in files:

        path = (
            REPO
            /
            relpath
        )


        if not path.is_file():

            continue


        record = {
            "repo_relative_path":
                relpath,

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }


        discovered[
            label
        ].append(
            record
        )


        print(
            f"  {record['sha256']} "
            f"{record['size_bytes']:10,d} B "
            f"{relpath}"
        )


    if not discovered[
        label
    ]:

        raise RuntimeError(
            f"No tracked artifacts found for {label}."
        )


# =============================================================================
# 11. RESOLVE EXACT STAGE26-4C3 SUMMARY
# =============================================================================

banner(
    "STAGE26-7A-R :: REPRESENTATION SUMMARY RESOLUTION"
)


rep_candidates = [
    record
    for record in discovered[
        "representation_4c3"
    ]
    if (
        "summary"
        in
        record[
            "repo_relative_path"
        ].lower()
    )
]


for record in rep_candidates:

    print(
        " ",
        record[
            "sha256"
        ],
        record[
            "repo_relative_path"
        ],
    )


rep_json_matches = [
    record
    for record in rep_candidates
    if (
        record[
            "sha256"
        ].startswith(
            "63c80411"
        )
        and
        Path(
            record[
                "repo_relative_path"
            ]
        ).suffix.lower()
        ==
        ".json"
    )
]


rep_csv_matches = [
    record
    for record in rep_candidates
    if (
        record[
            "sha256"
        ].startswith(
            "ca5bbc17"
        )
        and
        Path(
            record[
                "repo_relative_path"
            ]
        ).suffix.lower()
        ==
        ".csv"
    )
]


if len(
    rep_json_matches
) != 1:

    raise RuntimeError(
        "Could not uniquely resolve frozen 4C3 summary JSON."
    )


if len(
    rep_csv_matches
) != 1:

    raise RuntimeError(
        "Could not uniquely resolve frozen 4C3 summary CSV."
    )


representation_json = (
    REPO
    /
    rep_json_matches[
        0
    ][
        "repo_relative_path"
    ]
)

representation_csv = (
    REPO
    /
    rep_csv_matches[
        0
    ][
        "repo_relative_path"
    ]
)


print(
    "\nResolved representation JSON:"
)

print(
    " ",
    representation_json.relative_to(
        REPO
    )
)

print(
    " ",
    sha256_file(
        representation_json
    )
)


print(
    "\nResolved representation CSV:"
)

print(
    " ",
    representation_csv.relative_to(
        REPO
    )
)

print(
    " ",
    sha256_file(
        representation_csv
    )
)


# =============================================================================
# 12. EXACT REPRESENTATION THROUGHPUT FIELD
# =============================================================================

banner(
    "STAGE26-7A-R :: REPRESENTATION THROUGHPUT SCHEMA"
)


representation_rows = load_csv(
    representation_csv
)


if len(
    representation_rows
) != 5:

    raise RuntimeError(
        f"Expected five 4C3 summary rows; found {len(representation_rows)}."
    )


rep_columns = list(
    representation_rows[
        0
    ].keys()
)


print(
    "Columns:"
)

for column in rep_columns:

    print(
        " ",
        column
    )


rep_batches = sorted(
    int(
        row[
            "batch_size"
        ]
    )
    for row in representation_rows
)


if rep_batches != EXPECTED_BATCHES:

    raise RuntimeError(
        f"4C3 batch universe mismatch: {rep_batches}"
    )


throughput_fields = [
    column
    for column in rep_columns
    if (
        "throughput"
        in
        column.lower()
        or
        "flows_per_second"
        in
        column.lower()
    )
]


print(
    "\nThroughput-like columns:"
)

for column in throughput_fields:

    print(
        " ",
        column
    )


if not throughput_fields:

    raise RuntimeError(
        "No representation throughput field found."
    )


representation_inventory = []


for row in representation_rows:

    record = {
        "batch_size":
            int(
                row[
                    "batch_size"
                ]
            ),

        "throughput_fields": {
            field:
                row[
                    field
                ]
            for field in throughput_fields
        },

        "full_row":
            row,
    }


    representation_inventory.append(
        record
    )


    print(
        f"\nB={record['batch_size']}"
    )

    for field, value in record[
        "throughput_fields"
    ].items():

        print(
            " ",
            field,
            "=",
            value,
        )


# =============================================================================
# 13. RAW EXTRACTION 4B3 STRUCTURED EVIDENCE
# =============================================================================

banner(
    "STAGE26-7A-R :: RAW EXTRACTION COMPONENT EVIDENCE"
)


raw_extraction_evidence = []


for record in discovered[
    "raw_extraction_4b3"
]:

    path = (
        REPO
        /
        record[
            "repo_relative_path"
        ]
    )


    if path.suffix.lower() != ".json":

        continue


    try:

        obj = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )

    except Exception:

        continue


    evidence = scan_json_for_terms(
        obj,
        [
            "packets_per_second",
            "flows_per_second",
            "captured_mib",
            "elapsed",
            "throughput",
            "exportable",
            "lifecycle",
        ],
    )


    if evidence:

        raw_extraction_evidence.append(
            {
                "repo_relative_path":
                    record[
                        "repo_relative_path"
                    ],

                "sha256":
                    record[
                        "sha256"
                    ],

                "evidence":
                    evidence,
            }
        )


for artifact in raw_extraction_evidence:

    print(
        "\n",
        artifact[
            "repo_relative_path"
        ],
        sep="",
    )

    print(
        "  SHA256:",
        artifact[
            "sha256"
        ]
    )


    for item in artifact[
        "evidence"
    ][
        :40
    ]:

        print(
            " ",
            item[
                "path"
            ],
            "=",
            repr(
                item[
                    "value"
                ]
            ),
        )


if not raw_extraction_evidence:

    raise RuntimeError(
        "No structured 4B3 throughput evidence resolved."
    )


# =============================================================================
# 14. STAGE26-5A COMPOSABILITY EVIDENCE
# =============================================================================

banner(
    "STAGE26-7A-R :: STAGE26-5A COMPOSABILITY EVIDENCE"
)


e2e_evidence = []


for record in discovered[
    "e2e_closure_5a"
]:

    path = (
        REPO
        /
        record[
            "repo_relative_path"
        ]
    )


    if path.suffix.lower() != ".json":

        continue


    try:

        obj = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )

    except Exception:

        continue


    evidence = scan_json_for_terms(
        obj,
        [
            "complete_e2e",
            "additive",
            "prohibited",
            "available",
            "missing",
            "70_feature",
            "bridge",
            "imputed",
            "status",
        ],
    )


    if evidence:

        e2e_evidence.append(
            {
                "repo_relative_path":
                    record[
                        "repo_relative_path"
                    ],

                "sha256":
                    record[
                        "sha256"
                    ],

                "evidence":
                    evidence,
            }
        )


for artifact in e2e_evidence:

    print(
        "\n",
        artifact[
            "repo_relative_path"
        ],
        sep="",
    )


    for item in artifact[
        "evidence"
    ][
        :50
    ]:

        print(
            " ",
            item[
                "path"
            ],
            "=",
            repr(
                item[
                    "value"
                ]
            ),
        )


if not e2e_evidence:

    raise RuntimeError(
        "No Stage26-5A composability evidence resolved."
    )


# =============================================================================
# 15. MATCHED GROUP-B CPU1 BATCH AVAILABILITY
# =============================================================================

banner(
    "STAGE26-7A-R :: GROUP-B MATCHED CPU1 BATCH AVAILABILITY"
)


representation_batch_set = set(
    rep_batches
)


group_b_matched_batches = {}


for target in GROUP_B_TARGETS:

    inference_status = warm_geometry[
        target
    ][
        CPU1
    ]


    matched = []


    print(
        "\n",
        target,
        sep="",
    )


    for batch in EXPECTED_BATCHES:

        status_value = inference_status[
            batch
        ]


        representation_available = (
            batch
            in
            representation_batch_set
        )


        eligible = (
            representation_available
            and
            status_value
            ==
            "PASS"
        )


        record = {
            "batch_size":
                batch,

            "representation_available":
                representation_available,

            "inference_status":
                status_value,

            "component_ratio_eligible":
                eligible,
        }


        matched.append(
            record
        )


        print(
            f"  B={batch:5d} "
            f"representation={'YES' if representation_available else 'NO ':3s} "
            f"inference={status_value:24s} "
            f"ratio={'YES' if eligible else 'NO'}"
        )


    group_b_matched_batches[
        target
    ] = matched


# =============================================================================
# 16. PROSPECTIVE COMPATIBILITY MATRIX
# =============================================================================

banner(
    "STAGE26-7A-R :: PROSPECTIVE COMPATIBILITY MATRIX"
)


compatibility = [
    {
        "analysis_id":
            "INFERENCE_BATCH_SCALING_CPU1",

        "status":
            "ALLOWED",

        "scope":
            "ALL_8_STAGE26_INFERENCE_TARGETS",

        "hardware":
            CPU1,

        "matching_requirement":
            "WITHIN_TARGET_ACROSS_FROZEN_BATCH_SIZES",

        "claim":
            "INFERENCE_COMPONENT_BATCH_SCALING_ONLY",
    },

    {
        "analysis_id":
            "INFERENCE_BATCH_SCALING_CPU2",

        "status":
            "ALLOWED",

        "scope":
            "ALL_8_STAGE26_INFERENCE_TARGETS",

        "hardware":
            CPU2,

        "matching_requirement":
            "WITHIN_TARGET_ACROSS_FROZEN_BATCH_SIZES",

        "claim":
            "INFERENCE_COMPONENT_BATCH_SCALING_ONLY",
    },

    {
        "analysis_id":
            "GROUP_B_REPRESENTATION_VS_INFERENCE_CPU1",

        "status":
            "ALLOWED_COMPONENT_LEVEL_ONLY",

        "scope":
            GROUP_B_TARGETS,

        "hardware":
            CPU1,

        "matching_requirement":
            "EXACT_BATCH_MATCH_AND_BOTH_COMPONENTS_PASS",

        "ratio_definition":
            (
                "InferenceThroughput / RepresentationThroughput"
            ),

        "interpretation":
            (
                ">1 means representation has lower component capacity; "
                "<1 means inference has lower component capacity."
            ),

        "claim":
            (
                "DOWNSTREAM_COMPONENT_CAPACITY_RATIO_ONLY__"
                "NOT_COMPLETE_E2E"
            ),
    },

    {
        "analysis_id":
            "RAW_EXTRACTION_COMPONENT_REPORTING",

        "status":
            "ALLOWED_DESCRIPTIVE_ONLY",

        "claim":
            (
                "RAW_EXTRACTION_COMPONENT_THROUGHPUT_ONLY__"
                "NO_MODEL_PIPELINE_BOTTLENECK_INFERENCE"
            ),
    },

    {
        "analysis_id":
            "GROUP_A_EXTRACTION_VS_INFERENCE",

        "status":
            "PROHIBITED",

        "reason":
            "FROZEN_70_FEATURE_EXTRACTION_PATH_NOT_PROFILED",
    },

    {
        "analysis_id":
            "RAW_EXTRACTION_TO_GROUP_B_INFERENCE",

        "status":
            "PROHIBITED_AS_PIPELINE_RATIO",

        "reason":
            (
                "RAW_FLOW_TO_COMPACT_PACKET_IMAGE_CORPUS_BRIDGE_NOT_MEASURED"
            ),
    },

    {
        "analysis_id":
            "RAW_EXTRACTION_PLUS_REPRESENTATION_PLUS_INFERENCE_ADDITIVE_E2E",

        "status":
            "PROHIBITED",

        "reason":
            (
                "STAGE26_5A_CLOSED_NO_VALID_COMPLETE_E2E_MEASUREMENT"
            ),
    },

    {
        "analysis_id":
            "CROSS_GROUP_PARETO_FROM_CAPACITY",

        "status":
            "PROHIBITED",

        "reason":
            "NO_CROSS_GROUP_PARETO_FRONTIER",
    },
]


for item in compatibility:

    print(
        f"{item['analysis_id']:62s} "
        f"{item['status']}"
    )


# =============================================================================
# 17. WRITE TRANSIENT INVENTORY
# =============================================================================

banner(
    "STAGE26-7A-R :: WRITE TRANSIENT INVENTORY"
)


CAPACITY_RUNTIME.mkdir(
    parents=True,
    exist_ok=True,
)


payload = {
    "schema":
        "stage26_7a_capacity_source_compatibility_inventory_v2",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "status":
        "PASS_SOURCE_AND_COMPATIBILITY_RESOLUTION_BEFORE_CAPACITY_CALCULATION",

    "recovery_note": {
        "failed_cell_issue":
            (
                "Original Stage26-7A used non-existent hardware literal "
                "CPU_2_PHYSICAL_CORES."
            ),

        "frozen_actual_CPU2_literal":
            CPU2,

        "scientific_data_changed":
            False,
    },

    "hardware_modes": {
        "CPU1":
            CPU1,

        "CPU2":
            CPU2,
    },

    "known_source_identity":
        known_identity,

    "inference_geometry":
        warm_geometry,

    "corrected_uncertainty": {
        "PASS_conditions":
            73,

        "non_PASS_uncertainty_fabricated":
            0,

        "publication_uncertainty_source":
            str(
                CORRECTED_JSON.relative_to(
                    REPO
                )
            ),
    },

    "discovered_component_artifacts":
        discovered,

    "resolved_representation_sources": {
        "summary_json": {
            "repo_relative_path":
                str(
                    representation_json.relative_to(
                        REPO
                    )
                ),

            "sha256":
                sha256_file(
                    representation_json
                ),
        },

        "summary_csv": {
            "repo_relative_path":
                str(
                    representation_csv.relative_to(
                        REPO
                    )
                ),

            "sha256":
                sha256_file(
                    representation_csv
                ),
        },

        "throughput_fields":
            throughput_fields,
    },

    "representation_inventory":
        representation_inventory,

    "raw_extraction_structured_evidence":
        raw_extraction_evidence,

    "stage26_5a_composability_evidence":
        e2e_evidence,

    "group_b_matched_CPU1_batches":
        group_b_matched_batches,

    "compatibility_matrix":
        compatibility,

    "prospective_stage26_7_rules": {
        "CPU1_literal":
            CPU1,

        "CPU2_literal":
            CPU2,

        "batch_scaling_may_be_computed":
            True,

        "CPU1_and_CPU2_must_be_analyzed_separately":
            True,

        "OOM_timeout_preserved_as_resource_limits":
            True,

        "Stage26_6F1_corrected_uncertainty_is_publication_uncertainty_source":
            True,

        "historical_Stage26_2_CIs_are_publication_uncertainty_source":
            False,

        "Group_B_representation_inference_ratio_allowed":
            True,

        "Group_B_ratio_CPU1_only":
            True,

        "Group_B_ratio_requires_exact_batch_match":
            True,

        "Group_B_ratio_requires_both_components_available":
            True,

        "Group_B_ratio_is_complete_E2E":
            False,

        "Group_A_extraction_inference_ratio_allowed":
            False,

        "raw_extraction_to_Group_B_model_pipeline_ratio_allowed":
            False,

        "additive_complete_E2E_allowed":
            False,

        "missing_cost_imputation_allowed":
            False,

        "Pareto_membership_may_be_recomputed":
            False,

        "cross_group_Pareto_allowed":
            False,

        "GPU_allowed":
            False,
    },

    "scientific_state": {
        "capacity_ratio_computed":
            False,

        "batch_speedup_computed":
            False,

        "scaling_efficiency_computed":
            False,

        "bottleneck_declared":
            False,

        "complete_E2E_reconstructed":
            False,

        "missing_cost_imputed":
            False,

        "Pareto_recomputed":
            False,

        "bootstrap_executed":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "PCAP_accessed":
            False,

        "Release_corpus_accessed":
            False,

        "GPU_used":
            False,

        "Git_modified":
            False,
    },

    "next":
        (
            "Review exact 4C3 throughput field, 4B3 metric identities, "
            "Stage26-5A boundary evidence, and matched Group-B CPU1 batches. "
            "Then Git-freeze Stage26-7 sources and compatibility rules before "
            "computing any scaling or component-capacity ratio."
        ),
}


with OUT.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        payload,
        f,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    )

    f.write(
        "\n"
    )


out_sha = sha256_file(
    OUT
)


print(
    "Output:"
)

print(
    " ",
    OUT
)

print(
    "SHA256:"
)

print(
    " ",
    out_sha
)


# =============================================================================
# 18. FINAL READ-ONLY CLOSURE
# =============================================================================

banner(
    "STAGE26-7A-R SOURCE / COMPATIBILITY INVENTORY COMPLETE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed during Stage26-7A-R."
    )


if final_remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed during Stage26-7A-R."
    )


if final_status:

    raise RuntimeError(
        "Stage26-7A-R unexpectedly modified Git."
    )


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  CPU1 literal resolved                   :",
    CPU1
)

print(
    "  CPU2 literal resolved                   :",
    CPU2
)

print(
    "  exact five-batch geometry both modes    : YES"
)

print(
    "  PASS status/summary equality            : YES"
)

print(
    "  corrected uncertainty source resolved   : YES"
)

print(
    "  representation source resolved          : YES"
)

print(
    "  raw extraction evidence resolved        : YES"
)

print(
    "  Stage26-5A composability audited        : YES"
)

print(
    "  Group-B matched batches classified      : YES"
)

print(
    "  capacity ratio computed                 : NO"
)

print(
    "  batch scaling computed                  : NO"
)

print(
    "  bottleneck declared                     : NO"
)

print(
    "  complete E2E reconstructed              : NO"
)

print(
    "  missing cost imputed                    : NO"
)

print(
    "  Pareto changed                          : NO"
)

print(
    "  new inference                           : NO"
)

print(
    "  new timing                              : NO"
)

print(
    "  GPU                                     : NO"
)

print(
    "  Git modified                            : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Inspect the resolved 4C3 throughput field(s),"
)

print(
    "  4B3 component metrics, and Group-B matched batches."
)

print(
    "  If those are clean, Stage26-7B will Git-freeze this exact"
)

print(
    "  source/compatibility map BEFORE any ratio is calculated."
)


STAGE26-7A-R :: DURABLE SCIENTIFIC STATE
Expected parent: a484148cd8ea7d7c89604d3a015f73bd6f1bc81a
Local HEAD     : a484148cd8ea7d7c89604d3a015f73bd6f1bc81a
origin/main    : a484148cd8ea7d7c89604d3a015f73bd6f1bc81a
Repo clean     : True
Transient output exists: False

STAGE26-7A-R :: EXACT SOURCE IDENTITIES
warm raw                           PASS 78c58289ccfc4598966d6516201028f43c4b1d16d8bd0268a0cf58129d4179fa
warm summary                       PASS 75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232
warm status                        PASS df146563826992cb57702e589e4cf5d875ecfdbbf79533a731405e8eb738e7af
corrected uncertainty JSON         PASS 45ea4408738fb5101ce68a363c2cc0a7951648eed49e474da9489b00ce6128ce
corrected uncertainty CSV          PASS f804f117312b967bfc13cc070a34be43545238bfd30a758a00ad58202c9e6828
corrected uncertainty receipt      PASS 8d37b6bc2e0a7a75b8102b7d1cd3c1c058907420b229c9da1f302d97a44f1130
corrected uncertainty manifest     PASS 94f1acd338c933dbc7d4

In [26]:
# =============================================================================
# STAGE26-7B
# FREEZE CPU CAPACITY / BATCH-SCALING SOURCES, FORMULAS, AND COMPATIBILITY
# COMMIT + PUSH + REMOTE VERIFY
#
# DURABLE SCIENTIFIC PARENT:
#   a484148cd8ea7d7c89604d3a015f73bd6f1bc81a
#
# TRANSIENT STAGE26-7A INPUT:
#   /kaggle/working/stage26_deployment_profiling/capacity/
#       stage26_7a_capacity_source_compatibility_inventory.json
#
# EXPECTED SHA256:
#   3ecabd59e53f2f4784a154fdb8baa71fb2fa4504b41e4a7e91bf6b3e70d821c8
#
# PURPOSE
# -------
# Prospectively freeze Stage26-7 BEFORE any ratio, scaling factor,
# efficiency value, or bottleneck classification is computed.
#
# FROZEN ALLOWED ANALYSES
# -----------------------
#
# 1. WITHIN-TARGET INFERENCE BATCH SCALING
#
#    For target M, hardware H, batch B:
#
#        T(M,H,B) = frozen median_throughput_flows_per_second
#
#    Only PASS conditions are numerical.
#
#    B1-normalized throughput scaling:
#
#        batch_throughput_multiplier(M,H,B)
#            = T(M,H,B) / T(M,H,1)
#
#    B1-normalized amortized-latency improvement:
#
#        amortized_latency_improvement(M,H,B)
#            = L_amortized(M,H,1) / L_amortized(M,H,B)
#
#    where L_amortized is the frozen
#    median_amortized_latency_ms_per_flow.
#
#    These are INFERENCE-COMPONENT quantities only.
#
#
# 2. CPU2 VS CPU1 INFERENCE SCALING
#
#    Exact same target and batch only, both PASS:
#
#        two_core_speedup(M,B)
#            = T(M,CPU2,B) / T(M,CPU1,B)
#
#        two_core_parallel_efficiency(M,B)
#            = two_core_speedup(M,B) / 2
#
#    CPU1 = one physical core / one thread
#    CPU2 = two physical cores / two threads
#
#
# 3. GROUP-B REPRESENTATION VS INFERENCE COMPONENT CAPACITY
#
#    CPU1 only, exact same batch, both available:
#
#        component_capacity_ratio(M,B)
#            = inference_median_flows_per_second(M,B)
#              /
#              representation_median_flows_per_second(B)
#
#    Interpretation:
#
#        ratio > 1
#            representation component has lower measured capacity
#
#        ratio < 1
#            inference component has lower measured capacity
#
#        ratio = 1
#            equal measured component capacity
#
#    IMPORTANT:
#        This is a DOWNSTREAM COMPONENT-LEVEL ratio only.
#        It is NOT complete E2E throughput and NOT raw-PCAP-to-model capacity.
#
#
# 4. RAW EXTRACTION COMPONENT
#
#    Report descriptively only:
#
#        median exportable/reconstructed flows/s
#        median completed-lifecycle flows/s
#        median packets/s
#
#    Never algebraically combine these values with inference or
#    representation across the missing Stage26-5A boundaries.
#
#
# PROHIBITED
# ----------
# - Group-A extraction/inference pipeline ratio
# - raw extraction / Group-B inference pipeline ratio
# - min(component throughput) as complete-pipeline throughput
# - additive latency across missing boundaries
# - any complete raw-PCAP-to-model E2E reconstruction
# - missing-cost imputation
# - cross-group Pareto
# - Pareto membership recomputation from Stage26-7 capacity results
#
#
# UNCERTAINTY
# -----------
# Publication timing uncertainty source:
#   Stage26-6F1 corrected derived uncertainty.
#
# Historical Stage26-2 CIs:
#   audit-only, NOT protocol-certified.
#
# This checkpoint does NOT define bootstrap propagation for ratios.
# Ratio/scaling values remain point-estimate descriptive quantities unless a
# separate prospective uncertainty procedure is frozen later.
#
#
# 4C3 CAVEAT
# ----------
# The durable Stage26-4C3 representation measurement is preserved as measured.
# A known unresolved Stage26-8 audit item remains:
#
#   inter-iteration integrity/fingerprint reading outside the timed region may
#   perturb cache/CPU state before subsequent timed iterations.
#
# Therefore:
#   - Stage26-7 may use the current 4C3 point estimates as the durable measured
#     representation-component results;
#   - the component-capacity interpretation must retain this caveat;
#   - Stage26-8 must explicitly resolve/report the sensitivity issue;
#   - no 4C3 result is silently replaced here.
#
#
# THIS CELL DOES NOT:
# - compute ratios
# - compute scaling
# - compute efficiency
# - declare bottlenecks
# - execute bootstrap
# - rerun representation timing
# - run inference
# - run timing
# - load models
# - touch PCAP
# - touch Release corpus
# - recompute Pareto
# - use GPU
# =============================================================================

from __future__ import annotations

import os
import json
import stat
import hashlib
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "a484148cd8ea7d7c89604d3a015f73bd6f1bc81a"
)

COMMIT_SUBJECT = (
    "stage26: freeze CPU capacity compatibility and formulas"
)


RUNTIME_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

SOURCE_7A = (
    RUNTIME_ROOT
    / "capacity"
    / "stage26_7a_capacity_source_compatibility_inventory.json"
)

EXPECTED_7A_SHA256 = (
    "3ecabd59e53f2f4784a154fdb8baa71fb2fa4504b41e4a7e91bf6b3e70d821c8"
)


CPU1 = "CPU_1_PHYSICAL_CORE"
CPU2 = "CPU_2_PHYSICAL_CORE"

BATCHES = [
    1,
    64,
    256,
    1024,
    8192,
]


# =============================================================================
# 1. EXACT SOURCE ARTIFACTS
# =============================================================================

WARM_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_2_cpu_warm_inference"
)

WARM_SUMMARY = (
    WARM_DIR
    / "stage26_2_warm_summary.csv"
)

WARM_STATUS = (
    WARM_DIR
    / "stage26_2_condition_status.csv"
)

EXPECTED_WARM_SUMMARY_SHA256 = (
    "75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232"
)

EXPECTED_WARM_STATUS_SHA256 = (
    "df146563826992cb57702e589e4cf5d875ecfdbbf79533a731405e8eb738e7af"
)


UNCERTAINTY_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_6f1_bootstrap_corrected_uncertainty"
)

CORRECTED_UNCERTAINTY = (
    UNCERTAINTY_DIR
    / "stage26_6f1_bootstrap_corrected_uncertainty.json"
)

EXPECTED_CORRECTED_UNCERTAINTY_SHA256 = (
    "45ea4408738fb5101ce68a363c2cc0a7951648eed49e474da9489b00ce6128ce"
)


REP_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c3_cpu1_representation_timing"
)

REP_SUMMARY_CSV = (
    REP_DIR
    / "stage26_4c3_cpu1_representation_summary.csv"
)

REP_SUMMARY_JSON = (
    REP_DIR
    / "stage26_4c3_cpu1_representation_summary.json"
)

EXPECTED_REP_CSV_SHA256 = (
    "ca5bbc17edfa4bb088599236b4099c1986fd73cb7f087ef3ac753a524afd2ba5"
)

EXPECTED_REP_JSON_SHA256 = (
    "63c8041144668c62218fc9c46eb6a57aba9dcda0c9a3f0477f932f6a572c34f7"
)


EXTRACT_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4b3_cpu1_extraction_timing"
)

EXTRACT_SUMMARY = (
    EXTRACT_DIR
    / "stage26_4b3_cpu1_extraction_summary.json"
)

EXPECTED_EXTRACT_SUMMARY_SHA256 = (
    "1914504fad850879d1ed11afed432436c18454f2bb8650f881244c865f18ba16"
)


E2E_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_5a_e2e_availability_closure"
)

E2E_DECISION = (
    E2E_DIR
    / "stage26_5a_e2e_availability_decision.json"
)

E2E_RECEIPT = (
    E2E_DIR
    / "stage26_5a_e2e_availability_receipt.json"
)

EXPECTED_E2E_DECISION_SHA256 = (
    "be151f3e9891353eae63546b55bc621ab83bb8d4b829e9fefab0e29eafe4bc96"
)

EXPECTED_E2E_RECEIPT_SHA256 = (
    "d0948249d170114716e0c40b2b77f05f0106935366daf2ddff6e9c82fd58403c"
)


PARETO_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_6d_cpu_pareto_point_estimate"
)

PARETO_JSON = (
    PARETO_DIR
    / "stage26_6d_point_estimate_frontiers.json"
)

EXPECTED_PARETO_SHA256 = (
    "367b3d34ddb4dd125dcbe3db71291b5576b4521640cd2f11d76e50dc247fc6b5"
)


# =============================================================================
# 2. NEW CHECKPOINT
# =============================================================================

CHECKPOINT_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_7b_capacity_compatibility_lock"
)

CHECKPOINT_DIR = (
    REPO
    / CHECKPOINT_REL
)

SOURCE_LOCK = (
    CHECKPOINT_DIR
    / "stage26_7b_capacity_source_lock.json"
)

FORMULA_LOCK = (
    CHECKPOINT_DIR
    / "stage26_7b_capacity_formula_lock.json"
)

BOUNDARY_LOCK = (
    CHECKPOINT_DIR
    / "stage26_7b_capacity_boundary_lock.json"
)

FREEZE_RECEIPT = (
    CHECKPOINT_DIR
    / "stage26_7b_capacity_freeze_receipt.json"
)

MANIFEST = (
    CHECKPOINT_DIR
    / "stage26_7b_capacity_lock_manifest.json"
)


# =============================================================================
# 3. TARGETS / MATCHED GROUP-B BATCHES
# =============================================================================

ALL_TARGETS = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
    "ENS_LGBM_XGB_EQUAL",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
]


GROUP_B_MATCHED_CPU1 = {
    "STAGE20_MASKED_CNN_V1": [
        1,
        64,
        256,
    ],

    "STAGE21_MASKED_VIT_V1": [
        1,
        64,
        256,
        1024,
    ],
}


# =============================================================================
# 4. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            + " ".join(
                map(
                    str,
                    cmd,
                )
            )
            + "\n\n"
            + output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_json(
    path,
    payload,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        text=False,
    )

    return bytes(
        p.stdout
    )


# =============================================================================
# 5. DURABLE STATE
# =============================================================================

banner(
    "STAGE26-7B :: DURABLE SCIENTIFIC PARENT"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-7B parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Stage26-7B."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if CHECKPOINT_DIR.exists():

    raise RuntimeError(
        "Stage26-7B checkpoint already exists."
    )


# =============================================================================
# 6. TRANSIENT 7A IDENTITY
# =============================================================================

banner(
    "STAGE26-7B :: STAGE26-7A INPUT IDENTITY"
)


actual_7a_sha = sha256_file(
    SOURCE_7A
)


print(
    "Expected:",
    EXPECTED_7A_SHA256
)

print(
    "Actual  :",
    actual_7a_sha
)


if actual_7a_sha != EXPECTED_7A_SHA256:

    raise RuntimeError(
        "Stage26-7A transient inventory identity mismatch."
    )


inventory_7a = json.loads(
    SOURCE_7A.read_text(
        encoding="utf-8"
    )
)


if inventory_7a[
    "scientific_parent"
] != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-7A scientific parent mismatch."
    )


if inventory_7a[
    "scientific_state"
][
    "capacity_ratio_computed"
] is not False:

    raise RuntimeError(
        "Stage26-7A unexpectedly computed capacity ratios."
    )


if inventory_7a[
    "scientific_state"
][
    "batch_speedup_computed"
] is not False:

    raise RuntimeError(
        "Stage26-7A unexpectedly computed batch scaling."
    )


# =============================================================================
# 7. EXACT SOURCE HASH GATE
# =============================================================================

banner(
    "STAGE26-7B :: EXACT SOURCE HASH GATE"
)


source_checks = [
    (
        "warm summary",
        WARM_SUMMARY,
        EXPECTED_WARM_SUMMARY_SHA256,
    ),

    (
        "warm status",
        WARM_STATUS,
        EXPECTED_WARM_STATUS_SHA256,
    ),

    (
        "corrected uncertainty",
        CORRECTED_UNCERTAINTY,
        EXPECTED_CORRECTED_UNCERTAINTY_SHA256,
    ),

    (
        "representation summary CSV",
        REP_SUMMARY_CSV,
        EXPECTED_REP_CSV_SHA256,
    ),

    (
        "representation summary JSON",
        REP_SUMMARY_JSON,
        EXPECTED_REP_JSON_SHA256,
    ),

    (
        "raw extraction summary",
        EXTRACT_SUMMARY,
        EXPECTED_EXTRACT_SUMMARY_SHA256,
    ),

    (
        "E2E availability decision",
        E2E_DECISION,
        EXPECTED_E2E_DECISION_SHA256,
    ),

    (
        "E2E availability receipt",
        E2E_RECEIPT,
        EXPECTED_E2E_RECEIPT_SHA256,
    ),

    (
        "Pareto point-estimate result",
        PARETO_JSON,
        EXPECTED_PARETO_SHA256,
    ),
]


source_identity = {}


for label, path, expected in source_checks:

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:34s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Stage26-7B source identity mismatch: {label}"
        )


    source_identity[
        label
    ] = {
        "repo_relative_path":
            str(
                path.relative_to(
                    REPO
                )
            ),

        "sha256":
            actual,
    }


# =============================================================================
# 8. EXACT 5A BOUNDARY GATE
# =============================================================================

banner(
    "STAGE26-7B :: STAGE26-5A BOUNDARY GATE"
)


e2e_decision = json.loads(
    E2E_DECISION.read_text(
        encoding="utf-8"
    )
)

e2e_receipt = json.loads(
    E2E_RECEIPT.read_text(
        encoding="utf-8"
    )
)


if e2e_decision[
    "status"
] != "CLOSED_NO_VALID_COMPLETE_E2E_MEASUREMENT":

    raise RuntimeError(
        "Stage26-5A closure status changed."
    )


if e2e_decision[
    "group_A_dupsafe70"
][
    "complete_E2E_available"
] is not False:

    raise RuntimeError(
        "Group-A complete E2E unexpectedly available."
    )


if e2e_decision[
    "group_B_packet_image"
][
    "complete_raw_pcap_to_model_E2E_available"
] is not False:

    raise RuntimeError(
        "Group-B complete E2E unexpectedly available."
    )


if e2e_decision[
    "missing_measurement_is_not_imputed"
] is not True:

    raise RuntimeError(
        "Missing-cost imputation boundary changed."
    )


if e2e_decision[
    "prohibited_derivations"
][
    "full_pipeline_bottleneck_across_missing_boundaries"
] is not True:

    raise RuntimeError(
        "Full-pipeline bottleneck prohibition changed."
    )


if e2e_receipt[
    "group_A_70_feature_extraction_profiled"
] is not False:

    raise RuntimeError(
        "Group-A 70-feature extraction unexpectedly profiled."
    )


if e2e_receipt[
    "group_B_missing_raw_to_compact_bridge"
] is not True:

    raise RuntimeError(
        "Group-B missing bridge boundary changed."
    )


print(
    "Complete E2E available     : NO"
)

print(
    "Group-A feature extraction : NOT PROFILED"
)

print(
    "Group-B raw->compact bridge: MISSING"
)

print(
    "Missing-cost imputation    : FORBIDDEN"
)

print(
    "Full-pipeline bottleneck   : FORBIDDEN"
)


# =============================================================================
# 9. EXACT 4B3 COMPONENT SEMANTICS
# =============================================================================

banner(
    "STAGE26-7B :: RAW EXTRACTION METRIC SEMANTICS"
)


extract = json.loads(
    EXTRACT_SUMMARY.read_text(
        encoding="utf-8"
    )
)


metrics = extract[
    "metrics"
]


required_extract_metrics = [
    "packets_per_second",
    "flows_per_second",
    "completed_lifecycle_flows_per_second",
]


for metric in required_extract_metrics:

    if metric not in metrics:

        raise RuntimeError(
            f"Missing Stage26-4B3 metric: {metric}"
        )


extract_frozen_values = {
    "packets_per_second_median":
        float(
            metrics[
                "packets_per_second"
            ][
                "median"
            ]
        ),

    "exportable_reconstructed_flows_per_second_median":
        float(
            metrics[
                "flows_per_second"
            ][
                "median"
            ]
        ),

    "completed_lifecycle_flows_per_second_median":
        float(
            metrics[
                "completed_lifecycle_flows_per_second"
            ][
                "median"
            ]
        ),
}


print(
    "Median packets/s:"
)

print(
    " ",
    repr(
        extract_frozen_values[
            "packets_per_second_median"
        ]
    )
)


print(
    "Median exportable/reconstructed flows/s:"
)

print(
    " ",
    repr(
        extract_frozen_values[
            "exportable_reconstructed_flows_per_second_median"
        ]
    )
)


print(
    "Median completed-lifecycle flows/s:"
)

print(
    " ",
    repr(
        extract_frozen_values[
            "completed_lifecycle_flows_per_second_median"
        ]
    )
)


# =============================================================================
# 10. EXACT 4C3 REPRESENTATION SEMANTICS
# =============================================================================

banner(
    "STAGE26-7B :: REPRESENTATION METRIC SEMANTICS"
)


with REP_SUMMARY_CSV.open(
    "r",
    encoding="utf-8",
    newline="",
) as f:

    rep_rows = list(
        __import__(
            "csv"
        ).DictReader(
            f
        )
    )


if len(
    rep_rows
) != 5:

    raise RuntimeError(
        "Expected five Stage26-4C3 summary rows."
    )


rep_batches = sorted(
    int(
        row[
            "batch_size"
        ]
    )
    for row in rep_rows
)


if rep_batches != BATCHES:

    raise RuntimeError(
        "Stage26-4C3 batch universe changed."
    )


if "median_flows_per_second" not in rep_rows[
    0
]:

    raise RuntimeError(
        "Frozen representation median throughput field missing."
    )


representation_median_throughput = {
    int(
        row[
            "batch_size"
        ]
    ):
        float(
            row[
                "median_flows_per_second"
            ]
        )
    for row in rep_rows
}


for batch in BATCHES:

    print(
        f"B={batch:5d}: "
        f"{representation_median_throughput[batch]:.12f} flows/s"
    )


# =============================================================================
# 11. VERIFY GROUP-B MATCHED BATCH SET FROM 7A
# =============================================================================

banner(
    "STAGE26-7B :: GROUP-B MATCHED BATCH FREEZE"
)


actual_matched = {}


for target in GROUP_B_MATCHED_CPU1:

    rows = inventory_7a[
        "group_b_matched_CPU1_batches"
    ][
        target
    ]


    actual = [
        int(
            row[
                "batch_size"
            ]
        )
        for row in rows
        if row[
            "component_ratio_eligible"
        ]
        is True
    ]


    expected = GROUP_B_MATCHED_CPU1[
        target
    ]


    print(
        target
    )

    print(
        "  expected:",
        expected
    )

    print(
        "  actual  :",
        actual
    )


    if actual != expected:

        raise RuntimeError(
            f"{target}: matched batch set changed."
        )


    actual_matched[
        target
    ] = actual


# =============================================================================
# 12. CREATE CHECKPOINT
# =============================================================================

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# =============================================================================
# 13. WRITE SOURCE LOCK
# =============================================================================

source_lock = {
    "schema":
        "stage26_7b_capacity_source_lock_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-7B",

    "status":
        "FROZEN_BEFORE_CAPACITY_CALCULATION",

    "scientific_parent":
        EXPECTED_PARENT,

    "stage26_7a_transient": {
        "path":
            str(
                SOURCE_7A
            ),

        "sha256":
            EXPECTED_7A_SHA256,
    },

    "source_identity":
        source_identity,

    "hardware_modes": {
        "CPU1":
            {
                "literal":
                    CPU1,

                "physical_cores":
                    1,

                "threads":
                    1,
            },

        "CPU2":
            {
                "literal":
                    CPU2,

                "physical_cores":
                    2,

                "threads":
                    2,
            },
    },

    "frozen_batches":
        BATCHES,

    "inference_targets":
        ALL_TARGETS,

    "inference_point_fields": {
        "throughput":
            "median_throughput_flows_per_second",

        "amortized_latency":
            "median_amortized_latency_ms_per_flow",

        "condition_status_source":
            "stage26_2_condition_status.csv",

        "numerical_condition_requirement":
            "PASS_ONLY",
    },

    "inference_uncertainty_source": {
        "artifact":
            str(
                CORRECTED_UNCERTAINTY.relative_to(
                    REPO
                )
            ),

        "sha256":
            EXPECTED_CORRECTED_UNCERTAINTY_SHA256,

        "status":
            "PROTOCOL_CERTIFIED_DERIVED_UNCERTAINTY",

        "historical_stage26_2_CIs":
            "AUDIT_ONLY_NOT_PROTOCOL_CERTIFIED",
    },

    "representation": {
        "component":
            "STAGE20_COMPACT_CORPUS_TO_DENSE_UINT8_IMAGE_AND_BYTE_PADDING_MASK",

        "hardware":
            CPU1,

        "throughput_field":
            "median_flows_per_second",

        "throughput_unit":
            "flows_per_second",

        "summary_csv_sha256":
            EXPECTED_REP_CSV_SHA256,

        "summary_json_sha256":
            EXPECTED_REP_JSON_SHA256,

        "median_throughput_by_batch":
            representation_median_throughput,

        "eligible_Group_B_CPU1_batches":
            actual_matched,
    },

    "raw_extraction": {
        "component":
            "RAW_PCAP_TO_SOURCE_FAITHFUL_FLOW_RECONSTRUCTION_COMPONENT",

        "hardware":
            CPU1,

        "source_sha256":
            EXPECTED_EXTRACT_SUMMARY_SHA256,

        "median_component_metrics":
            extract_frozen_values,

        "semantic_note":
            (
                "flows_per_second is exportable/reconstructed-flow rate; "
                "completed_lifecycle_flows_per_second is a distinct completed-"
                "lifecycle rate. Neither is a frozen model-input throughput."
            ),
    },
}


atomic_json(
    SOURCE_LOCK,
    source_lock,
)


# =============================================================================
# 14. WRITE FORMULA LOCK
# =============================================================================

formula_lock = {
    "schema":
        "stage26_7b_capacity_formula_lock_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-7B",

    "status":
        "FROZEN_BEFORE_ANY_RATIO_OR_SCALING_RESULT",

    "batch_scaling": {
        "allowed":
            True,

        "scope":
            "WITHIN_TARGET_WITHIN_HARDWARE_MODE",

        "reference_batch":
            1,

        "throughput_multiplier": {
            "formula":
                "T(target,hardware,batch) / T(target,hardware,1)",

            "T":
                "median_throughput_flows_per_second",

            "requirement":
                "BOTH_BATCH_AND_B1_PASS",

            "claim":
                "INFERENCE_COMPONENT_BATCH_SCALING_ONLY",
        },

        "amortized_latency_improvement": {
            "formula":
                (
                    "L_amortized(target,hardware,1) / "
                    "L_amortized(target,hardware,batch)"
                ),

            "L_amortized":
                "median_amortized_latency_ms_per_flow",

            "requirement":
                "BOTH_BATCH_AND_B1_PASS",

            "claim":
                "INFERENCE_COMPONENT_ONLY",
        },
    },

    "two_core_scaling": {
        "allowed":
            True,

        "matching":
            "EXACT_TARGET_AND_BATCH",

        "requirement":
            "CPU1_AND_CPU2_BOTH_PASS",

        "speedup": {
            "formula":
                "T(target,CPU2,batch) / T(target,CPU1,batch)",
        },

        "parallel_efficiency": {
            "formula":
                "two_core_speedup / 2.0",

            "denominator_reason":
                "CPU2 uses two physical cores versus one physical core.",
        },

        "claim":
            "INFERENCE_COMPONENT_CPU_SCALING_ONLY",
    },

    "group_B_representation_inference_capacity": {
        "allowed":
            True,

        "hardware":
            CPU1,

        "matching":
            "EXACT_BATCH",

        "eligible_batches":
            actual_matched,

        "ratio": {
            "formula":
                (
                    "inference_median_throughput_flows_per_second / "
                    "representation_median_flows_per_second"
                ),

            "name":
                "component_capacity_ratio",
        },

        "interpretation": {
            "ratio_gt_1":
                (
                    "REPRESENTATION_COMPONENT_LOWER_MEASURED_CAPACITY"
                ),

            "ratio_lt_1":
                "INFERENCE_COMPONENT_LOWER_MEASURED_CAPACITY",

            "ratio_eq_1":
                "EQUAL_MEASURED_COMPONENT_CAPACITY",
        },

        "claim_boundary":
            (
                "DOWNSTREAM_COMPONENT_LEVEL_ONLY__"
                "NOT_COMPLETE_E2E__NOT_RAW_PCAP_TO_MODEL"
            ),
    },

    "raw_extraction": {
        "ratio_to_model_inference_allowed":
            False,

        "ratio_to_representation_allowed":
            False,

        "descriptive_metrics_allowed": [
            "packets_per_second_median",
            "exportable_reconstructed_flows_per_second_median",
            "completed_lifecycle_flows_per_second_median",
        ],
    },

    "uncertainty": {
        "corrected_stage26_6f1_timing_uncertainty_may_be_reported":
            True,

        "historical_stage26_2_CIs_may_be_used_for_publication":
            False,

        "ratio_uncertainty_propagation_frozen":
            False,

        "ratio_uncertainty_to_be_invented_post_hoc":
            False,

        "ratio_and_scaling_results":
            "DESCRIPTIVE_POINT_ESTIMATES_UNLESS_SEPARATE_PROSPECTIVE_METHOD_FROZEN",
    },
}


atomic_json(
    FORMULA_LOCK,
    formula_lock,
)


# =============================================================================
# 15. WRITE BOUNDARY / CAVEAT LOCK
# =============================================================================

boundary_lock = {
    "schema":
        "stage26_7b_capacity_boundary_lock_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-7B",

    "status":
        "FROZEN_BEFORE_CAPACITY_INTERPRETATION",

    "stage26_5a": {
        "complete_E2E_available":
            False,

        "group_A_70_feature_extraction_profiled":
            False,

        "group_B_raw_to_compact_bridge_missing":
            True,

        "missing_measurement_imputation_allowed":
            False,
    },

    "prohibited_derivations": [
        "GROUP_A_EXTRACTION_TO_INFERENCE_PIPELINE_RATIO",
        "RAW_EXTRACTION_TO_GROUP_B_INFERENCE_PIPELINE_RATIO",
        "RAW_EXTRACTION_TO_REPRESENTATION_PIPELINE_RATIO",
        "MINIMUM_COMPONENT_THROUGHPUT_AS_COMPLETE_PIPELINE_THROUGHPUT",
        "SUM_COMPONENT_LATENCY_AS_COMPLETE_E2E",
        "FULL_PIPELINE_BOTTLENECK_ACROSS_MISSING_BOUNDARIES",
        "CROSS_GROUP_EXTRACTION_EQUIVALENCE",
        "CROSS_GROUP_PARETO_FROM_CAPACITY",
        "PARETO_MEMBERSHIP_RECOMPUTATION_FROM_STAGE26_7",
    ],

    "group_B_component_ratio": {
        "allowed":
            True,

        "complete_E2E":
            False,

        "raw_pcap_to_model":
            False,

        "predictive_comparison":
            False,

        "claim":
            "COMPONENT_CAPACITY_COMPARISON_ONLY",
    },

    "stage26_4c3_measurement_caveat": {
        "status":
            "OPEN_STAGE26_8_AUDIT_ITEM",

        "durable_measurement_retained":
            True,

        "issue":
            (
                "The Stage26-4C3 implementation performs full-array output "
                "integrity/fingerprint reading outside the timed region between "
                "timed iterations. Although excluded from each timer window, "
                "those reads may perturb cache/CPU state before subsequent "
                "timed iterations."
            ),

        "stage26_7_policy":
            (
                "Use the durable 4C3 point estimates as measured, retain the "
                "caveat in component-capacity interpretation, and do not "
                "silently replace or rerun the measurement."
            ),

        "stage26_8_required_action":
            (
                "Explicitly audit/report the inter-iteration integrity-read "
                "sensitivity and, if a prospective cleaner implementation "
                "sensitivity run is performed, preserve 4C3 as the original "
                "measurement rather than overwriting it."
            ),
    },

    "pareto": {
        "stage26_6d_sha256":
            EXPECTED_PARETO_SHA256,

        "membership_immutable":
            True,

        "cross_group_frontier":
            False,
    },

    "GPU_allowed":
        False,
}


atomic_json(
    BOUNDARY_LOCK,
    boundary_lock,
)


# =============================================================================
# 16. RECEIPT
# =============================================================================

source_lock_sha = sha256_file(
    SOURCE_LOCK
)

formula_lock_sha = sha256_file(
    FORMULA_LOCK
)

boundary_lock_sha = sha256_file(
    BOUNDARY_LOCK
)


freeze_receipt = {
    "schema":
        "stage26_7b_capacity_freeze_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-7B",

    "status":
        "PASS_CAPACITY_SOURCES_FORMULAS_BOUNDARIES_FROZEN_BEFORE_CALCULATION",

    "scientific_parent":
        EXPECTED_PARENT,

    "stage26_7a_sha256":
        EXPECTED_7A_SHA256,

    "locks": {
        "source_lock_sha256":
            source_lock_sha,

        "formula_lock_sha256":
            formula_lock_sha,

        "boundary_lock_sha256":
            boundary_lock_sha,
    },

    "frozen_analysis_permissions": {
        "inference_batch_scaling":
            True,

        "CPU2_vs_CPU1_inference_scaling":
            True,

        "Group_B_representation_vs_inference_component_ratio":
            True,

        "raw_extraction_descriptive_reporting":
            True,

        "Group_A_extraction_inference_pipeline_ratio":
            False,

        "raw_extraction_Group_B_pipeline_ratio":
            False,

        "complete_E2E_reconstruction":
            False,

        "missing_cost_imputation":
            False,

        "Pareto_recomputation":
            False,

        "cross_group_Pareto":
            False,
    },

    "scientific_state": {
        "capacity_ratio_computed":
            False,

        "batch_scaling_computed":
            False,

        "two_core_speedup_computed":
            False,

        "parallel_efficiency_computed":
            False,

        "bottleneck_declared":
            False,

        "ratio_uncertainty_computed":
            False,

        "new_bootstrap_executed":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "complete_E2E_reconstructed":
            False,

        "missing_cost_imputed":
            False,

        "Pareto_recomputed":
            False,

        "PCAP_accessed":
            False,

        "Release_corpus_accessed":
            False,

        "GPU_used":
            False,
    },

    "next_permitted_action":
        (
            "After remote Git verification, compute only the Stage26-7B "
            "frozen descriptive point-estimate scaling and scientifically "
            "compatible Group-B component-capacity ratios."
        ),
}


atomic_json(
    FREEZE_RECEIPT,
    freeze_receipt,
)


receipt_sha = sha256_file(
    FREEZE_RECEIPT
)


# =============================================================================
# 17. MANIFEST
# =============================================================================

package_files = [
    SOURCE_LOCK,
    FORMULA_LOCK,
    BOUNDARY_LOCK,
    FREEZE_RECEIPT,
]


manifest_rows = []


for path in package_files:

    manifest_rows.append(
        {
            "repo_relative_path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest = {
    "schema":
        "stage26_7b_capacity_lock_manifest_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-7B",

    "status":
        "READY_FOR_GIT_ANCHOR",

    "scientific_parent":
        EXPECTED_PARENT,

    "commit_subject":
        COMMIT_SUBJECT,

    "stage26_7a_sha256":
        EXPECTED_7A_SHA256,

    "source_lock_sha256":
        source_lock_sha,

    "formula_lock_sha256":
        formula_lock_sha,

    "boundary_lock_sha256":
        boundary_lock_sha,

    "freeze_receipt_sha256":
        receipt_sha,

    "file_count_excluding_manifest":
        len(
            manifest_rows
        ),

    "files":
        manifest_rows,

    "scientific_state": {
        "sources_frozen":
            True,

        "formulas_frozen":
            True,

        "boundaries_frozen":
            True,

        "capacity_values_computed":
            False,

        "new_measurement_performed":
            False,

        "Pareto_changed":
            False,

        "GPU_used":
            False,
    },
}


atomic_json(
    MANIFEST,
    manifest,
)


manifest_sha = sha256_file(
    MANIFEST
)


# =============================================================================
# 18. PRINT LOCK SUMMARY
# =============================================================================

banner(
    "STAGE26-7B :: LOCK SUMMARY"
)


print(
    "Source lock  :",
    source_lock_sha
)

print(
    "Formula lock :",
    formula_lock_sha
)

print(
    "Boundary lock:",
    boundary_lock_sha
)

print(
    "Receipt      :",
    receipt_sha
)

print(
    "Manifest     :",
    manifest_sha
)


print(
    "\nGROUP-B MATCHED CPU1 BATCHES:"
)

for target, batches in actual_matched.items():

    print(
        " ",
        target,
        batches
    )


print(
    "\nRAW EXTRACTION — DESCRIPTIVE ONLY:"
)

for key, value in extract_frozen_values.items():

    print(
        " ",
        key,
        "=",
        repr(
            value
        )
    )


# =============================================================================
# 19. LOCAL PACKAGE AUDIT
# =============================================================================

banner(
    "STAGE26-7B :: LOCAL PACKAGE AUDIT"
)


for row in manifest_rows:

    path = (
        REPO
        /
        row[
            "repo_relative_path"
        ]
    )

    size = int(
        path.stat().st_size
    )

    digest = sha256_file(
        path
    )

    passed = (
        size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        digest
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{size:10,d} B "
        f"{digest} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Local Stage26-7B package audit failed."
        )


# =============================================================================
# 20. GIT CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-7B :: GIT CHANGE AUDIT"
)


repo_status = git(
    "status",
    "--porcelain",
)


print(
    repo_status
)


if not repo_status:

    raise RuntimeError(
        "Expected uncommitted Stage26-7B package."
    )


unexpected = []


for line in repo_status.splitlines():

    relpath = line[
        3:
    ]


    if not relpath.startswith(
        str(
            CHECKPOINT_REL
        )
        +
        "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository changes:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 21. GIT IDENTITY
# =============================================================================

author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_PARENT,
)

author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_PARENT,
)


if (
    not author_name.strip()
    or
    "@"
    not in
    author_email
):

    raise RuntimeError(
        "Could not recover Git identity."
    )


git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


print(
    "\nGit author:"
)

print(
    " ",
    author_name,
    "<" + author_email + ">"
)


# =============================================================================
# 22. COMMIT
# =============================================================================

banner(
    "STAGE26-7B :: COMMIT"
)


git(
    "add",
    str(
        CHECKPOINT_REL
    ),
)


print(
    git(
        "diff",
        "--cached",
        "--name-status",
    )
)


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-7B commit parent mismatch."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Stage26-7B commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository not clean after Stage26-7B commit."
    )


# =============================================================================
# 23. PUSH
# =============================================================================

banner(
    "STAGE26-7B :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    push_result = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
        text=True,
    )


    print(
        push_result.stdout.strip()
    )


github_token = None


# =============================================================================
# 24. REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-7B :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "origin/main did not advance to Stage26-7B."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote Stage26-7B parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote Stage26-7B subject mismatch."
    )


# =============================================================================
# 25. REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-7B :: REMOTE BYTE VERIFICATION"
)


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    str(
        MANIFEST.relative_to(
            REPO
        )
    ),
)


remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "Remote manifest SHA256:"
)

print(
    " ",
    remote_manifest_sha
)


if remote_manifest_sha != manifest_sha:

    raise RuntimeError(
        "Remote Stage26-7B manifest mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


for row in remote_manifest[
    "files"
]:

    data = git_blob_bytes(
        "origin/main",
        row[
            "repo_relative_path"
        ],
    )

    actual_size = len(
        data
    )

    actual_sha = sha256_bytes(
        data
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Remote Stage26-7B byte verification failed."
        )


# =============================================================================
# 26. REMOTE SCIENTIFIC VERIFICATION
# =============================================================================

banner(
    "STAGE26-7B :: REMOTE SCIENTIFIC VERIFICATION"
)


remote_source = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            SOURCE_LOCK.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_formula = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            FORMULA_LOCK.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_boundary = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            BOUNDARY_LOCK.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_receipt = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            FREEZE_RECEIPT.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)


remote_checks = {
    "frozen before calculation":
        (
            remote_source[
                "status"
            ]
            ==
            "FROZEN_BEFORE_CAPACITY_CALCULATION"
        ),

    "CPU1 literal":
        (
            remote_source[
                "hardware_modes"
            ][
                "CPU1"
            ][
                "literal"
            ]
            ==
            CPU1
        ),

    "CPU2 literal":
        (
            remote_source[
                "hardware_modes"
            ][
                "CPU2"
            ][
                "literal"
            ]
            ==
            CPU2
        ),

    "representation median field":
        (
            remote_source[
                "representation"
            ][
                "throughput_field"
            ]
            ==
            "median_flows_per_second"
        ),

    "CNN matched batches":
        (
            remote_source[
                "representation"
            ][
                "eligible_Group_B_CPU1_batches"
            ][
                "STAGE20_MASKED_CNN_V1"
            ]
            ==
            [
                1,
                64,
                256,
            ]
        ),

    "ViT matched batches":
        (
            remote_source[
                "representation"
            ][
                "eligible_Group_B_CPU1_batches"
            ][
                "STAGE21_MASKED_VIT_V1"
            ]
            ==
            [
                1,
                64,
                256,
                1024,
            ]
        ),

    "component ratio only":
        (
            remote_formula[
                "group_B_representation_inference_capacity"
            ][
                "claim_boundary"
            ]
            ==
            (
                "DOWNSTREAM_COMPONENT_LEVEL_ONLY__"
                "NOT_COMPLETE_E2E__NOT_RAW_PCAP_TO_MODEL"
            )
        ),

    "complete E2E false":
        (
            remote_boundary[
                "stage26_5a"
            ][
                "complete_E2E_available"
            ]
            is False
        ),

    "missing imputation false":
        (
            remote_boundary[
                "stage26_5a"
            ][
                "missing_measurement_imputation_allowed"
            ]
            is False
        ),

    "4C3 caveat retained":
        (
            remote_boundary[
                "stage26_4c3_measurement_caveat"
            ][
                "status"
            ]
            ==
            "OPEN_STAGE26_8_AUDIT_ITEM"
        ),

    "capacity not computed":
        (
            remote_receipt[
                "scientific_state"
            ][
                "capacity_ratio_computed"
            ]
            is False
        ),

    "batch scaling not computed":
        (
            remote_receipt[
                "scientific_state"
            ][
                "batch_scaling_computed"
            ]
            is False
        ),

    "Pareto not recomputed":
        (
            remote_receipt[
                "scientific_state"
            ][
                "Pareto_recomputed"
            ]
            is False
        ),

    "GPU false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "GPU_used"
            ]
            is False
        ),
}


for name, passed in remote_checks.items():

    print(
        f"{name:38s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    remote_checks.values()
):

    raise RuntimeError(
        "Remote Stage26-7B scientific verification failed."
    )


# =============================================================================
# 27. FINAL CLOSURE
# =============================================================================

banner(
    "STAGE26-7B CPU CAPACITY / SCALING FREEZE COMPLETE"
)


final_status = git(
    "status",
    "--porcelain",
)


print(
    "NEW DURABLE COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nPARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nFROZEN ALLOWED ANALYSES:"
)

print(
    "  inference batch scaling          : YES"
)

print(
    "  CPU2 vs CPU1 scaling             : YES"
)

print(
    "  Group-B representation/inference : YES — component level only"
)

print(
    "  raw extraction reporting         : YES — descriptive only"
)


print(
    "\nPROHIBITED:"
)

print(
    "  Group-A extraction/inference E2E : YES"
)

print(
    "  raw extraction -> Group-B model  : YES"
)

print(
    "  complete E2E reconstruction      : YES"
)

print(
    "  missing-cost imputation          : YES"
)

print(
    "  cross-group Pareto               : YES"
)

print(
    "  Pareto recomputation             : YES"
)


print(
    "\nGROUP-B MATCHED CPU1 BATCHES:"
)

print(
    "  CNN:",
    actual_matched[
        "STAGE20_MASKED_CNN_V1"
    ]
)

print(
    "  ViT:",
    actual_matched[
        "STAGE21_MASKED_VIT_V1"
    ]
)


print(
    "\n4C3 CAVEAT:"
)

print(
    "  inter-iteration integrity-read sensitivity remains OPEN"
)

print(
    "  and is explicitly deferred to Stage26-8 audit."
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  sources frozen             : YES"
)

print(
    "  formulas frozen            : YES"
)

print(
    "  boundaries frozen          : YES"
)

print(
    "  capacity ratio computed    : NO"
)

print(
    "  batch scaling computed     : NO"
)

print(
    "  CPU scaling computed       : NO"
)

print(
    "  bottleneck declared        : NO"
)

print(
    "  new bootstrap              : NO"
)

print(
    "  new inference              : NO"
)

print(
    "  new timing                 : NO"
)

print(
    "  complete E2E reconstructed : NO"
)

print(
    "  Pareto changed             : NO"
)

print(
    "  GPU                        : NO"
)


print(
    "\nHASHES:"
)

print(
    "  source lock  :",
    source_lock_sha
)

print(
    "  formula lock :",
    formula_lock_sha
)

print(
    "  boundary lock:",
    boundary_lock_sha
)

print(
    "  receipt      :",
    receipt_sha
)

print(
    "  manifest     :",
    manifest_sha
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  commit             : PASS"
)

print(
    "  parent             : PASS"
)

print(
    "  package bytes      : PASS"
)

print(
    "  source semantics   : PASS"
)

print(
    "  formula semantics  : PASS"
)

print(
    "  boundary semantics : PASS"
)


print(
    "\nRepo clean:",
    final_status == ""
)


if final_status:

    raise RuntimeError(
        "Repository not clean after Stage26-7B."
    )


print(
    "\nNEXT:"
)

print(
    "  Stage26-7C may compute ONLY the frozen descriptive"
)

print(
    "  inference scaling and compatible Group-B component-capacity"
)

print(
    "  ratios from these remotely anchored sources/formulas."
)

print(
    "  No complete E2E or cross-boundary bottleneck result is permitted."
)

print(
    "  No GPU yet."
)


STAGE26-7B :: DURABLE SCIENTIFIC PARENT
Expected parent: a484148cd8ea7d7c89604d3a015f73bd6f1bc81a
Local HEAD     : a484148cd8ea7d7c89604d3a015f73bd6f1bc81a
origin/main    : a484148cd8ea7d7c89604d3a015f73bd6f1bc81a
Repo clean     : True

STAGE26-7B :: STAGE26-7A INPUT IDENTITY
Expected: 3ecabd59e53f2f4784a154fdb8baa71fb2fa4504b41e4a7e91bf6b3e70d821c8
Actual  : 3ecabd59e53f2f4784a154fdb8baa71fb2fa4504b41e4a7e91bf6b3e70d821c8

STAGE26-7B :: EXACT SOURCE HASH GATE
warm summary                       PASS 75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232
warm status                        PASS df146563826992cb57702e589e4cf5d875ecfdbbf79533a731405e8eb738e7af
corrected uncertainty              PASS 45ea4408738fb5101ce68a363c2cc0a7951648eed49e474da9489b00ce6128ce
representation summary CSV         PASS ca5bbc17edfa4bb088599236b4099c1986fd73cb7f087ef3ac753a524afd2ba5
representation summary JSON        PASS 63c8041144668c62218fc9c46eb6a57aba9dcda0c9a3f0477f932f6a572c34f7
raw extra

In [27]:
# =============================================================================
# STAGE26-7C
# COMPUTE FROZEN CPU INFERENCE SCALING AND SCIENTIFICALLY COMPATIBLE
# GROUP-B REPRESENTATION/INFERENCE COMPONENT CAPACITY RESULTS
# COMMIT + PUSH + REMOTE VERIFY
#
# DURABLE SCIENTIFIC PARENT:
#   c524e763b8ae7593204f95f23f9e819b4dcaaf9f
#
# STAGE26-7B LOCKS:
#   source lock   c56a08cc88a2e8d75df0832b72e41ecbb90ee1cf3f12c0a7402d9fc1d5247c42
#   formula lock  7cb33c57dd030e6fb2c0763eec70471b4afc0f9d06d071bc690fbac02d6dacea
#   boundary lock 1def25c4046770dd1da2ff7939c38b01bc19917bbecf01423c9cad684b68221c
#   receipt       0bf2b935e8b5509b4fa6d32413a9ce835af7ead4166a77f173f553f940369790
#   manifest      f3f844c1fb3953831cd345f5ac767730e0933efb068ad9b7a79e50ccfe5789e8
#
# FROZEN CALCULATIONS
# -------------------
#
# A. Within-target / within-hardware batch scaling:
#
#       throughput_multiplier
#           = median_throughput(B) / median_throughput(B1)
#
#       amortized_latency_improvement
#           = median_amortized_latency(B1)
#             / median_amortized_latency(B)
#
#    Numerical results only for PASS conditions.
#
#
# B. CPU2 vs CPU1, exact target+batch, both PASS:
#
#       two_core_speedup
#           = throughput_CPU2 / throughput_CPU1
#
#       two_core_parallel_efficiency
#           = two_core_speedup / 2
#
#
# C. Group-B CPU1 representation/inference component capacity:
#
#       ratio
#           = inference_median_flows_per_second
#             / representation_median_flows_per_second
#
#    Eligible:
#
#       CNN : B=[1,64,256]
#       ViT : B=[1,64,256,1024]
#
#    Interpretation:
#
#       ratio > 1:
#           representation component has lower measured capacity
#
#       ratio < 1:
#           inference component has lower measured capacity
#
#    This is COMPONENT-LEVEL ONLY.
#    It is NOT a full-pipeline bottleneck result.
#
#
# D. Raw extraction:
#
#    Report frozen component rates descriptively only.
#    They are NEVER used in a pipeline ratio here.
#
#
# UNCERTAINTY
# -----------
# Stage26-6F1 remains the publication timing uncertainty source.
#
# No uncertainty propagation is performed for:
#   - throughput multipliers
#   - CPU2 speedup
#   - parallel efficiency
#   - representation/inference capacity ratios
#
# These Stage26-7C quantities are DESCRIPTIVE POINT ESTIMATES.
#
#
# 4C3 CAVEAT
# ----------
# Group-B component-capacity results retain:
#
#   OPEN_STAGE26_8_AUDIT_ITEM
#
# because the original 4C3 representation timing has the known
# inter-iteration integrity-read/cache-state sensitivity concern.
#
#
# ABSOLUTELY NO:
#   - complete E2E reconstruction
#   - raw extraction / model ratio
#   - Group-A extraction / inference ratio
#   - missing-cost imputation
#   - pipeline bottleneck claim
#   - Pareto recomputation
#   - cross-group Pareto
#   - bootstrap
#   - model loading
#   - inference
#   - timing
#   - PCAP
#   - Release corpus
#   - GPU
# =============================================================================

from __future__ import annotations

import csv
import json
import os
import stat
import math
import hashlib
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "c524e763b8ae7593204f95f23f9e819b4dcaaf9f"
)

COMMIT_SUBJECT = (
    "stage26: anchor CPU capacity and scaling results"
)


CPU1 = "CPU_1_PHYSICAL_CORE"
CPU2 = "CPU_2_PHYSICAL_CORE"


BATCHES = [
    1,
    64,
    256,
    1024,
    8192,
]


ALL_TARGETS = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
    "ENS_LGBM_XGB_EQUAL",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
]


GROUP_B = [
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
]


# =============================================================================
# 1. STAGE26-7B LOCKS
# =============================================================================

LOCK_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_7b_capacity_compatibility_lock"
)

SOURCE_LOCK = (
    LOCK_DIR
    / "stage26_7b_capacity_source_lock.json"
)

FORMULA_LOCK = (
    LOCK_DIR
    / "stage26_7b_capacity_formula_lock.json"
)

BOUNDARY_LOCK = (
    LOCK_DIR
    / "stage26_7b_capacity_boundary_lock.json"
)

LOCK_RECEIPT = (
    LOCK_DIR
    / "stage26_7b_capacity_freeze_receipt.json"
)

LOCK_MANIFEST = (
    LOCK_DIR
    / "stage26_7b_capacity_lock_manifest.json"
)


EXPECTED_SOURCE_LOCK_SHA256 = (
    "c56a08cc88a2e8d75df0832b72e41ecbb90ee1cf3f12c0a7402d9fc1d5247c42"
)

EXPECTED_FORMULA_LOCK_SHA256 = (
    "7cb33c57dd030e6fb2c0763eec70471b4afc0f9d06d071bc690fbac02d6dacea"
)

EXPECTED_BOUNDARY_LOCK_SHA256 = (
    "1def25c4046770dd1da2ff7939c38b01bc19917bbecf01423c9cad684b68221c"
)

EXPECTED_LOCK_RECEIPT_SHA256 = (
    "0bf2b935e8b5509b4fa6d32413a9ce835af7ead4166a77f173f553f940369790"
)

EXPECTED_LOCK_MANIFEST_SHA256 = (
    "f3f844c1fb3953831cd345f5ac767730e0933efb068ad9b7a79e50ccfe5789e8"
)


# =============================================================================
# 2. FROZEN SOURCE ARTIFACTS
# =============================================================================

WARM_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_2_cpu_warm_inference"
)

WARM_SUMMARY = (
    WARM_DIR
    / "stage26_2_warm_summary.csv"
)

WARM_STATUS = (
    WARM_DIR
    / "stage26_2_condition_status.csv"
)


EXPECTED_WARM_SUMMARY_SHA256 = (
    "75a60850faf85e10a96a3d9865bb196f4a8cc50111255a2cc15248c54d646232"
)

EXPECTED_WARM_STATUS_SHA256 = (
    "df146563826992cb57702e589e4cf5d875ecfdbbf79533a731405e8eb738e7af"
)


REP_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c3_cpu1_representation_timing"
)

REP_SUMMARY_CSV = (
    REP_DIR
    / "stage26_4c3_cpu1_representation_summary.csv"
)

EXPECTED_REP_CSV_SHA256 = (
    "ca5bbc17edfa4bb088599236b4099c1986fd73cb7f087ef3ac753a524afd2ba5"
)


EXTRACT_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4b3_cpu1_extraction_timing"
)

EXTRACT_SUMMARY = (
    EXTRACT_DIR
    / "stage26_4b3_cpu1_extraction_summary.json"
)

EXPECTED_EXTRACT_SUMMARY_SHA256 = (
    "1914504fad850879d1ed11afed432436c18454f2bb8650f881244c865f18ba16"
)


UNCERTAINTY_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_6f1_bootstrap_corrected_uncertainty"
)

CORRECTED_UNCERTAINTY = (
    UNCERTAINTY_DIR
    / "stage26_6f1_bootstrap_corrected_uncertainty.json"
)

EXPECTED_CORRECTED_UNCERTAINTY_SHA256 = (
    "45ea4408738fb5101ce68a363c2cc0a7951648eed49e474da9489b00ce6128ce"
)


PARETO_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_6d_cpu_pareto_point_estimate"
)

PARETO_JSON = (
    PARETO_DIR
    / "stage26_6d_point_estimate_frontiers.json"
)

EXPECTED_PARETO_SHA256 = (
    "367b3d34ddb4dd125dcbe3db71291b5576b4521640cd2f11d76e50dc247fc6b5"
)


# =============================================================================
# 3. OUTPUT PACKAGE
# =============================================================================

CHECKPOINT_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_7c_cpu_capacity_scaling"
)

CHECKPOINT_DIR = (
    REPO
    / CHECKPOINT_REL
)


BATCH_SCALING_CSV = (
    CHECKPOINT_DIR
    / "stage26_7c_inference_batch_scaling.csv"
)

TWO_CORE_CSV = (
    CHECKPOINT_DIR
    / "stage26_7c_two_core_scaling.csv"
)

GROUP_B_CAPACITY_CSV = (
    CHECKPOINT_DIR
    / "stage26_7c_group_b_component_capacity.csv"
)

RAW_EXTRACTION_JSON = (
    CHECKPOINT_DIR
    / "stage26_7c_raw_extraction_descriptive.json"
)

RESULTS_JSON = (
    CHECKPOINT_DIR
    / "stage26_7c_capacity_scaling_results.json"
)

RECEIPT = (
    CHECKPOINT_DIR
    / "stage26_7c_capacity_scaling_receipt.json"
)

MANIFEST = (
    CHECKPOINT_DIR
    / "stage26_7c_capacity_scaling_manifest.json"
)


# =============================================================================
# 4. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            + " ".join(
                map(
                    str,
                    cmd,
                )
            )
            + "\n\n"
            + output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def load_csv(path):

    with Path(path).open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def atomic_json(
    path,
    payload,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        text=False,
    )

    return bytes(
        p.stdout
    )


def positive_float(
    value,
    label,
):

    x = float(
        value
    )

    if (
        not math.isfinite(
            x
        )
        or
        x <= 0
    ):

        raise RuntimeError(
            f"{label}: expected finite positive value; got {value!r}"
        )

    return x


# =============================================================================
# 5. DURABLE GIT GATE
# =============================================================================

banner(
    "STAGE26-7C :: DURABLE SCIENTIFIC PARENT"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-7C parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Stage26-7C."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if CHECKPOINT_DIR.exists():

    raise RuntimeError(
        "Stage26-7C checkpoint already exists."
    )


# =============================================================================
# 6. EXACT LOCK IDENTITY
# =============================================================================

banner(
    "STAGE26-7C :: EXACT STAGE26-7B LOCK IDENTITY"
)


lock_checks = [
    (
        "source lock",
        SOURCE_LOCK,
        EXPECTED_SOURCE_LOCK_SHA256,
    ),

    (
        "formula lock",
        FORMULA_LOCK,
        EXPECTED_FORMULA_LOCK_SHA256,
    ),

    (
        "boundary lock",
        BOUNDARY_LOCK,
        EXPECTED_BOUNDARY_LOCK_SHA256,
    ),

    (
        "freeze receipt",
        LOCK_RECEIPT,
        EXPECTED_LOCK_RECEIPT_SHA256,
    ),

    (
        "lock manifest",
        LOCK_MANIFEST,
        EXPECTED_LOCK_MANIFEST_SHA256,
    ),
]


for label, path, expected in lock_checks:

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:24s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Stage26-7B lock mismatch: {label}"
        )


# =============================================================================
# 7. LOCK SEMANTICS GATE
# =============================================================================

banner(
    "STAGE26-7C :: LOCKED FORMULA / BOUNDARY SEMANTICS"
)


source_lock = json.loads(
    SOURCE_LOCK.read_text(
        encoding="utf-8"
    )
)

formula_lock = json.loads(
    FORMULA_LOCK.read_text(
        encoding="utf-8"
    )
)

boundary_lock = json.loads(
    BOUNDARY_LOCK.read_text(
        encoding="utf-8"
    )
)

lock_receipt = json.loads(
    LOCK_RECEIPT.read_text(
        encoding="utf-8"
    )
)


if source_lock[
    "status"
] != "FROZEN_BEFORE_CAPACITY_CALCULATION":

    raise RuntimeError(
        "Stage26-7B source lock was not frozen before calculation."
    )


if formula_lock[
    "status"
] != "FROZEN_BEFORE_ANY_RATIO_OR_SCALING_RESULT":

    raise RuntimeError(
        "Stage26-7B formula lock status changed."
    )


if boundary_lock[
    "stage26_5a"
][
    "complete_E2E_available"
] is not False:

    raise RuntimeError(
        "Complete E2E unexpectedly available."
    )


if boundary_lock[
    "stage26_5a"
][
    "missing_measurement_imputation_allowed"
] is not False:

    raise RuntimeError(
        "Missing-cost imputation unexpectedly allowed."
    )


if boundary_lock[
    "pareto"
][
    "membership_immutable"
] is not True:

    raise RuntimeError(
        "Pareto membership is not frozen."
    )


if boundary_lock[
    "GPU_allowed"
] is not False:

    raise RuntimeError(
        "GPU unexpectedly permitted."
    )


if lock_receipt[
    "scientific_state"
][
    "capacity_ratio_computed"
] is not False:

    raise RuntimeError(
        "Capacity ratios were already computed before Stage26-7C."
    )


if lock_receipt[
    "scientific_state"
][
    "batch_scaling_computed"
] is not False:

    raise RuntimeError(
        "Batch scaling was already computed before Stage26-7C."
    )


print(
    "Batch scaling                 : ALLOWED"
)

print(
    "CPU2 vs CPU1 scaling          : ALLOWED"
)

print(
    "Group-B component ratio       : ALLOWED — component level only"
)

print(
    "Raw extraction pipeline ratio : PROHIBITED"
)

print(
    "Complete E2E                  : PROHIBITED"
)

print(
    "Missing-cost imputation       : PROHIBITED"
)

print(
    "Pareto recomputation          : PROHIBITED"
)

print(
    "GPU                           : PROHIBITED"
)


# =============================================================================
# 8. EXACT SOURCE HASH GATE
# =============================================================================

banner(
    "STAGE26-7C :: EXACT SOURCE HASH GATE"
)


source_checks = [
    (
        "warm summary",
        WARM_SUMMARY,
        EXPECTED_WARM_SUMMARY_SHA256,
    ),

    (
        "warm status",
        WARM_STATUS,
        EXPECTED_WARM_STATUS_SHA256,
    ),

    (
        "representation summary",
        REP_SUMMARY_CSV,
        EXPECTED_REP_CSV_SHA256,
    ),

    (
        "raw extraction summary",
        EXTRACT_SUMMARY,
        EXPECTED_EXTRACT_SUMMARY_SHA256,
    ),

    (
        "corrected uncertainty",
        CORRECTED_UNCERTAINTY,
        EXPECTED_CORRECTED_UNCERTAINTY_SHA256,
    ),

    (
        "Pareto result",
        PARETO_JSON,
        EXPECTED_PARETO_SHA256,
    ),
]


for label, path, expected in source_checks:

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:28s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Stage26-7C source mismatch: {label}"
        )


# =============================================================================
# 9. LOAD FROZEN INFERENCE SOURCES
# =============================================================================

banner(
    "STAGE26-7C :: INFERENCE SOURCE GEOMETRY"
)


status_rows = load_csv(
    WARM_STATUS
)

summary_rows = load_csv(
    WARM_SUMMARY
)


if len(
    status_rows
) != 80:

    raise RuntimeError(
        "Expected 80 condition-status rows."
    )


if len(
    summary_rows
) != 73:

    raise RuntimeError(
        "Expected 73 PASS summary rows."
    )


status_by_key = {
    (
        row[
            "target_id"
        ],
        row[
            "hardware_mode"
        ],
        int(
            row[
                "batch_size"
            ]
        ),
    ):
        row
    for row in status_rows
}


summary_by_key = {
    (
        row[
            "target_id"
        ],
        row[
            "hardware_mode"
        ],
        int(
            row[
                "batch_size"
            ]
        ),
    ):
        row
    for row in summary_rows
}


if len(
    status_by_key
) != 80:

    raise RuntimeError(
        "Condition key collision in Stage26-2 status."
    )


if len(
    summary_by_key
) != 73:

    raise RuntimeError(
        "Condition key collision in Stage26-2 summary."
    )


for target in ALL_TARGETS:

    for hardware in [
        CPU1,
        CPU2,
    ]:

        for batch in BATCHES:

            key = (
                target,
                hardware,
                batch,
            )


            if key not in status_by_key:

                raise RuntimeError(
                    f"Missing frozen condition: {key}"
                )


            status_value = status_by_key[
                key
            ][
                "status"
            ]


            if status_value == "PASS":

                if key not in summary_by_key:

                    raise RuntimeError(
                        f"PASS condition missing summary: {key}"
                    )

            else:

                if key in summary_by_key:

                    raise RuntimeError(
                        f"Non-PASS condition has numeric summary: {key}"
                    )


print(
    "Status conditions :",
    len(
        status_by_key
    )
)

print(
    "PASS summaries    :",
    len(
        summary_by_key
    )
)

print(
    "Geometry audit    : PASS"
)


# =============================================================================
# 10. CREATE OUTPUT DIRECTORY
# =============================================================================

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# =============================================================================
# 11. A — WITHIN-TARGET INFERENCE BATCH SCALING
# =============================================================================

banner(
    "STAGE26-7C :: A — INFERENCE BATCH SCALING"
)


batch_scaling_rows = []


for target in ALL_TARGETS:

    for hardware in [
        CPU1,
        CPU2,
    ]:

        b1_key = (
            target,
            hardware,
            1,
        )


        if status_by_key[
            b1_key
        ][
            "status"
        ] != "PASS":

            raise RuntimeError(
                f"{target}/{hardware}: B1 reference is not PASS."
            )


        b1_summary = summary_by_key[
            b1_key
        ]


        b1_throughput = positive_float(
            b1_summary[
                "median_throughput_flows_per_second"
            ],
            f"{target}/{hardware}/B1 throughput",
        )


        b1_amortized = positive_float(
            b1_summary[
                "median_amortized_latency_ms_per_flow"
            ],
            f"{target}/{hardware}/B1 amortized latency",
        )


        for batch in BATCHES:

            key = (
                target,
                hardware,
                batch,
            )

            condition = status_by_key[
                key
            ]

            status_value = condition[
                "status"
            ]


            if status_value != "PASS":

                batch_scaling_rows.append(
                    {
                        "target_id":
                            target,

                        "hardware_mode":
                            hardware,

                        "batch_size":
                            batch,

                        "condition_id":
                            condition[
                                "condition_id"
                            ],

                        "status":
                            status_value,

                        "median_throughput_flows_per_second":
                            "",

                        "median_amortized_latency_ms_per_flow":
                            "",

                        "throughput_multiplier_vs_B1":
                            "",

                        "amortized_latency_improvement_vs_B1":
                            "",

                        "claim_boundary":
                            "INFERENCE_COMPONENT_ONLY",

                        "uncertainty_propagated":
                            False,
                    }
                )

                continue


            summary = summary_by_key[
                key
            ]


            throughput = positive_float(
                summary[
                    "median_throughput_flows_per_second"
                ],
                f"{target}/{hardware}/B{batch} throughput",
            )


            amortized = positive_float(
                summary[
                    "median_amortized_latency_ms_per_flow"
                ],
                f"{target}/{hardware}/B{batch} amortized latency",
            )


            throughput_multiplier = (
                throughput
                /
                b1_throughput
            )


            amortized_improvement = (
                b1_amortized
                /
                amortized
            )


            if batch == 1:

                if throughput_multiplier != 1.0:

                    raise RuntimeError(
                        f"{target}/{hardware}: B1 throughput multiplier != 1."
                    )


                if amortized_improvement != 1.0:

                    raise RuntimeError(
                        f"{target}/{hardware}: B1 latency improvement != 1."
                    )


            batch_scaling_rows.append(
                {
                    "target_id":
                        target,

                    "hardware_mode":
                        hardware,

                    "batch_size":
                        batch,

                    "condition_id":
                        condition[
                            "condition_id"
                        ],

                    "status":
                        status_value,

                    "median_throughput_flows_per_second":
                        throughput,

                    "median_amortized_latency_ms_per_flow":
                        amortized,

                    "throughput_multiplier_vs_B1":
                        throughput_multiplier,

                    "amortized_latency_improvement_vs_B1":
                        amortized_improvement,

                    "claim_boundary":
                        "INFERENCE_COMPONENT_ONLY",

                    "uncertainty_propagated":
                        False,
                }
            )


if len(
    batch_scaling_rows
) != 80:

    raise RuntimeError(
        "Expected 80 batch-scaling condition rows."
    )


batch_numeric_rows = [
    row
    for row in batch_scaling_rows
    if row[
        "status"
    ]
    ==
    "PASS"
]


if len(
    batch_numeric_rows
) != 73:

    raise RuntimeError(
        "Expected 73 numerical batch-scaling rows."
    )


print(
    "Rows total     :",
    len(
        batch_scaling_rows
    )
)

print(
    "Numerical PASS :",
    len(
        batch_numeric_rows
    )
)

print(
    "Resource-limit :",
    len(
        batch_scaling_rows
    )
    -
    len(
        batch_numeric_rows
    )
)


batch_fields = [
    "target_id",
    "hardware_mode",
    "batch_size",
    "condition_id",
    "status",
    "median_throughput_flows_per_second",
    "median_amortized_latency_ms_per_flow",
    "throughput_multiplier_vs_B1",
    "amortized_latency_improvement_vs_B1",
    "claim_boundary",
    "uncertainty_propagated",
]


with BATCH_SCALING_CSV.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=batch_fields,
    )

    writer.writeheader()

    writer.writerows(
        batch_scaling_rows
    )


# =============================================================================
# 12. B — CPU2 VS CPU1 SCALING
# =============================================================================

banner(
    "STAGE26-7C :: B — CPU2 VS CPU1 SCALING"
)


two_core_rows = []


for target in ALL_TARGETS:

    for batch in BATCHES:

        key1 = (
            target,
            CPU1,
            batch,
        )

        key2 = (
            target,
            CPU2,
            batch,
        )


        status1 = status_by_key[
            key1
        ][
            "status"
        ]

        status2 = status_by_key[
            key2
        ][
            "status"
        ]


        eligible = (
            status1 == "PASS"
            and
            status2 == "PASS"
        )


        if not eligible:

            continue


        t1 = positive_float(
            summary_by_key[
                key1
            ][
                "median_throughput_flows_per_second"
            ],
            f"{target}/CPU1/B{batch}",
        )

        t2 = positive_float(
            summary_by_key[
                key2
            ][
                "median_throughput_flows_per_second"
            ],
            f"{target}/CPU2/B{batch}",
        )


        speedup = (
            t2
            /
            t1
        )


        efficiency = (
            speedup
            /
            2.0
        )


        two_core_rows.append(
            {
                "target_id":
                    target,

                "batch_size":
                    batch,

                "CPU1_condition_id":
                    status_by_key[
                        key1
                    ][
                        "condition_id"
                    ],

                "CPU2_condition_id":
                    status_by_key[
                        key2
                    ][
                        "condition_id"
                    ],

                "CPU1_median_throughput_flows_per_second":
                    t1,

                "CPU2_median_throughput_flows_per_second":
                    t2,

                "two_core_speedup":
                    speedup,

                "two_core_parallel_efficiency":
                    efficiency,

                "claim_boundary":
                    "INFERENCE_COMPONENT_CPU_SCALING_ONLY",

                "uncertainty_propagated":
                    False,
            }
        )


expected_two_core_count = 0


for target in ALL_TARGETS:

    for batch in BATCHES:

        if (
            status_by_key[
                (
                    target,
                    CPU1,
                    batch,
                )
            ][
                "status"
            ]
            ==
            "PASS"
            and
            status_by_key[
                (
                    target,
                    CPU2,
                    batch,
                )
            ][
                "status"
            ]
            ==
            "PASS"
        ):

            expected_two_core_count += 1


if len(
    two_core_rows
) != expected_two_core_count:

    raise RuntimeError(
        "CPU2/CPU1 matched-condition count mismatch."
    )


print(
    "Matched PASS target/batch pairs:",
    len(
        two_core_rows
    )
)


two_core_fields = [
    "target_id",
    "batch_size",
    "CPU1_condition_id",
    "CPU2_condition_id",
    "CPU1_median_throughput_flows_per_second",
    "CPU2_median_throughput_flows_per_second",
    "two_core_speedup",
    "two_core_parallel_efficiency",
    "claim_boundary",
    "uncertainty_propagated",
]


with TWO_CORE_CSV.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=two_core_fields,
    )

    writer.writeheader()

    writer.writerows(
        two_core_rows
    )


# =============================================================================
# 13. C — GROUP-B REPRESENTATION / INFERENCE COMPONENT CAPACITY
# =============================================================================

banner(
    "STAGE26-7C :: C — GROUP-B COMPONENT CAPACITY"
)


rep_rows = load_csv(
    REP_SUMMARY_CSV
)


representation_by_batch = {
    int(
        row[
            "batch_size"
        ]
    ):
        positive_float(
            row[
                "median_flows_per_second"
            ],
            f"representation/B{row['batch_size']}",
        )
    for row in rep_rows
}


locked_eligible = source_lock[
    "representation"
][
    "eligible_Group_B_CPU1_batches"
]


component_rows = []


for target in GROUP_B:

    eligible_batches = [
        int(
            value
        )
        for value in locked_eligible[
            target
        ]
    ]


    for batch in eligible_batches:

        key = (
            target,
            CPU1,
            batch,
        )


        if status_by_key[
            key
        ][
            "status"
        ] != "PASS":

            raise RuntimeError(
                f"Locked Group-B matched condition no longer PASS: {key}"
            )


        inference_throughput = positive_float(
            summary_by_key[
                key
            ][
                "median_throughput_flows_per_second"
            ],
            f"{target}/CPU1/B{batch} inference throughput",
        )


        representation_throughput = representation_by_batch[
            batch
        ]


        ratio = (
            inference_throughput
            /
            representation_throughput
        )


        if ratio > 1.0:

            direction = (
                "REPRESENTATION_COMPONENT_LOWER_MEASURED_CAPACITY"
            )

        elif ratio < 1.0:

            direction = (
                "INFERENCE_COMPONENT_LOWER_MEASURED_CAPACITY"
            )

        else:

            direction = (
                "EQUAL_MEASURED_COMPONENT_CAPACITY"
            )


        component_rows.append(
            {
                "target_id":
                    target,

                "hardware_mode":
                    CPU1,

                "batch_size":
                    batch,

                "inference_condition_id":
                    status_by_key[
                        key
                    ][
                        "condition_id"
                    ],

                "representation_median_flows_per_second":
                    representation_throughput,

                "inference_median_flows_per_second":
                    inference_throughput,

                "component_capacity_ratio_inference_over_representation":
                    ratio,

                "lower_measured_capacity_component":
                    direction,

                "claim_boundary":
                    (
                        "DOWNSTREAM_COMPONENT_LEVEL_ONLY__"
                        "NOT_COMPLETE_E2E__NOT_RAW_PCAP_TO_MODEL"
                    ),

                "representation_measurement_caveat":
                    "OPEN_STAGE26_8_AUDIT_ITEM",

                "uncertainty_propagated":
                    False,
            }
        )


if len(
    component_rows
) != 7:

    raise RuntimeError(
        f"Expected exactly seven compatible Group-B ratios; "
        f"found {len(component_rows)}."
    )


component_fields = [
    "target_id",
    "hardware_mode",
    "batch_size",
    "inference_condition_id",
    "representation_median_flows_per_second",
    "inference_median_flows_per_second",
    "component_capacity_ratio_inference_over_representation",
    "lower_measured_capacity_component",
    "claim_boundary",
    "representation_measurement_caveat",
    "uncertainty_propagated",
]


with GROUP_B_CAPACITY_CSV.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=component_fields,
    )

    writer.writeheader()

    writer.writerows(
        component_rows
    )


for row in component_rows:

    print(
        f"{row['target_id']:34s} "
        f"B={row['batch_size']:4d} "
        f"rep={row['representation_median_flows_per_second']:12.3f} "
        f"infer={row['inference_median_flows_per_second']:12.3f} "
        f"ratio={row['component_capacity_ratio_inference_over_representation']:.6f} "
        f"{row['lower_measured_capacity_component']}"
    )


# =============================================================================
# 14. D — RAW EXTRACTION DESCRIPTIVE ONLY
# =============================================================================

banner(
    "STAGE26-7C :: D — RAW EXTRACTION DESCRIPTIVE ONLY"
)


extract_locked = source_lock[
    "raw_extraction"
]


raw_descriptive = {
    "schema":
        "stage26_7c_raw_extraction_descriptive_v1",

    "status":
        "COMPONENT_ONLY_NOT_COMPOSED",

    "component":
        extract_locked[
            "component"
        ],

    "hardware":
        extract_locked[
            "hardware"
        ],

    "source_sha256":
        extract_locked[
            "source_sha256"
        ],

    "median_component_metrics":
        extract_locked[
            "median_component_metrics"
        ],

    "semantic_note":
        extract_locked[
            "semantic_note"
        ],

    "used_in_pipeline_ratio":
        False,

    "used_in_complete_E2E":
        False,

    "bottleneck_claim":
        False,
}


atomic_json(
    RAW_EXTRACTION_JSON,
    raw_descriptive,
)


for key, value in raw_descriptive[
    "median_component_metrics"
].items():

    print(
        f"{key:56s}: "
        f"{value}"
    )


print(
    "\nRaw extraction used in model-pipeline ratio: NO"
)


# =============================================================================
# 15. COMPACT RESULT JSON
# =============================================================================

banner(
    "STAGE26-7C :: RESULT PACKAGE"
)


results_payload = {
    "schema":
        "stage26_7c_capacity_scaling_results_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-7C",

    "status":
        "PASS_FROZEN_DESCRIPTIVE_CPU_CAPACITY_AND_SCALING",

    "scientific_parent":
        EXPECTED_PARENT,

    "stage26_7b_locks": {
        "source_lock_sha256":
            EXPECTED_SOURCE_LOCK_SHA256,

        "formula_lock_sha256":
            EXPECTED_FORMULA_LOCK_SHA256,

        "boundary_lock_sha256":
            EXPECTED_BOUNDARY_LOCK_SHA256,

        "freeze_receipt_sha256":
            EXPECTED_LOCK_RECEIPT_SHA256,

        "manifest_sha256":
            EXPECTED_LOCK_MANIFEST_SHA256,
    },

    "batch_scaling": {
        "condition_rows":
            len(
                batch_scaling_rows
            ),

        "PASS_numeric_rows":
            len(
                batch_numeric_rows
            ),

        "resource_limit_rows":
            (
                len(
                    batch_scaling_rows
                )
                -
                len(
                    batch_numeric_rows
                )
            ),

        "reference_batch":
            1,

        "throughput_multiplier_formula":
            "T(batch)/T(B1)",

        "amortized_latency_improvement_formula":
            "L_amortized(B1)/L_amortized(batch)",

        "claim_boundary":
            "INFERENCE_COMPONENT_ONLY",

        "uncertainty_propagated":
            False,
    },

    "two_core_scaling": {
        "matched_PASS_pairs":
            len(
                two_core_rows
            ),

        "speedup_formula":
            "T(CPU2)/T(CPU1)",

        "parallel_efficiency_formula":
            "speedup/2",

        "claim_boundary":
            "INFERENCE_COMPONENT_CPU_SCALING_ONLY",

        "uncertainty_propagated":
            False,
    },

    "group_B_component_capacity": {
        "ratio_count":
            len(
                component_rows
            ),

        "rows":
            component_rows,

        "ratio_formula":
            (
                "inference_median_flows_per_second / "
                "representation_median_flows_per_second"
            ),

        "claim_boundary":
            (
                "DOWNSTREAM_COMPONENT_LEVEL_ONLY__"
                "NOT_COMPLETE_E2E__NOT_RAW_PCAP_TO_MODEL"
            ),

        "representation_measurement_caveat":
            "OPEN_STAGE26_8_AUDIT_ITEM",

        "uncertainty_propagated":
            False,
    },

    "raw_extraction": {
        "reported_descriptively":
            True,

        "used_in_ratio":
            False,

        "used_in_complete_E2E":
            False,

        "metrics":
            raw_descriptive[
                "median_component_metrics"
            ],
    },

    "publication_uncertainty": {
        "timing_uncertainty_source":
            str(
                CORRECTED_UNCERTAINTY.relative_to(
                    REPO
                )
            ),

        "timing_uncertainty_sha256":
            EXPECTED_CORRECTED_UNCERTAINTY_SHA256,

        "ratio_uncertainty_propagated":
            False,

        "scaling_uncertainty_propagated":
            False,

        "reason":
            (
                "Stage26-7B froze Stage26-7 scaling and capacity ratios "
                "as descriptive point estimates; no ratio uncertainty "
                "procedure was prospectively frozen."
            ),
    },

    "scientific_boundaries": {
        "complete_E2E_available":
            False,

        "complete_E2E_reconstructed":
            False,

        "Group_A_extraction_inference_ratio":
            False,

        "raw_extraction_Group_B_pipeline_ratio":
            False,

        "missing_cost_imputed":
            False,

        "full_pipeline_bottleneck_declared":
            False,

        "component_capacity_direction_classified":
            True,

        "Pareto_recomputed":
            False,

        "Pareto_changed":
            False,

        "cross_group_Pareto":
            False,

        "new_bootstrap":
            False,

        "new_measurement":
            False,

        "GPU":
            False,
    },
}


atomic_json(
    RESULTS_JSON,
    results_payload,
)


# =============================================================================
# 16. RECEIPT
# =============================================================================

batch_sha = sha256_file(
    BATCH_SCALING_CSV
)

two_core_sha = sha256_file(
    TWO_CORE_CSV
)

component_sha = sha256_file(
    GROUP_B_CAPACITY_CSV
)

raw_sha = sha256_file(
    RAW_EXTRACTION_JSON
)

results_sha = sha256_file(
    RESULTS_JSON
)


receipt_payload = {
    "schema":
        "stage26_7c_capacity_scaling_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-7C",

    "status":
        "PASS_CPU_CAPACITY_AND_SCALING_WITH_FROZEN_BOUNDARIES",

    "scientific_parent":
        EXPECTED_PARENT,

    "frozen_lock_identity": {
        "source_lock_sha256":
            EXPECTED_SOURCE_LOCK_SHA256,

        "formula_lock_sha256":
            EXPECTED_FORMULA_LOCK_SHA256,

        "boundary_lock_sha256":
            EXPECTED_BOUNDARY_LOCK_SHA256,

        "freeze_receipt_sha256":
            EXPECTED_LOCK_RECEIPT_SHA256,

        "lock_manifest_sha256":
            EXPECTED_LOCK_MANIFEST_SHA256,
    },

    "results": {
        "batch_scaling_total_rows":
            len(
                batch_scaling_rows
            ),

        "batch_scaling_numeric_PASS_rows":
            len(
                batch_numeric_rows
            ),

        "two_core_matched_PASS_rows":
            len(
                two_core_rows
            ),

        "Group_B_component_capacity_rows":
            len(
                component_rows
            ),
    },

    "outputs": {
        "batch_scaling_csv_sha256":
            batch_sha,

        "two_core_scaling_csv_sha256":
            two_core_sha,

        "group_B_component_capacity_csv_sha256":
            component_sha,

        "raw_extraction_descriptive_sha256":
            raw_sha,

        "results_json_sha256":
            results_sha,
    },

    "scientific_state": {
        "batch_scaling_computed":
            True,

        "two_core_speedup_computed":
            True,

        "parallel_efficiency_computed":
            True,

        "Group_B_component_capacity_ratio_computed":
            True,

        "Group_B_component_capacity_direction_classified":
            True,

        "ratio_uncertainty_computed":
            False,

        "scaling_uncertainty_computed":
            False,

        "new_bootstrap_executed":
            False,

        "raw_extraction_used_in_ratio":
            False,

        "complete_E2E_reconstructed":
            False,

        "missing_cost_imputed":
            False,

        "full_pipeline_bottleneck_declared":
            False,

        "Pareto_recomputed":
            False,

        "Pareto_changed":
            False,

        "cross_group_Pareto":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "PCAP_accessed":
            False,

        "Release_corpus_accessed":
            False,

        "GPU_used":
            False,
    },

    "stage26_4c3_caveat":
        "OPEN_STAGE26_8_AUDIT_ITEM",

    "next":
        (
            "Proceed to Stage26-8 CPU closure/audit. Explicitly resolve the "
            "Stage26-4C3 inter-iteration integrity-read sensitivity before "
            "final publication framing and before enabling GPU."
        ),
}


atomic_json(
    RECEIPT,
    receipt_payload,
)


receipt_sha = sha256_file(
    RECEIPT
)


# =============================================================================
# 17. MANIFEST
# =============================================================================

package_files = [
    BATCH_SCALING_CSV,
    TWO_CORE_CSV,
    GROUP_B_CAPACITY_CSV,
    RAW_EXTRACTION_JSON,
    RESULTS_JSON,
    RECEIPT,
]


manifest_rows = []


for path in package_files:

    manifest_rows.append(
        {
            "repo_relative_path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest_payload = {
    "schema":
        "stage26_7c_capacity_scaling_manifest_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-7C",

    "status":
        "READY_FOR_GIT_ANCHOR",

    "scientific_parent":
        EXPECTED_PARENT,

    "commit_subject":
        COMMIT_SUBJECT,

    "stage26_7b_source_lock_sha256":
        EXPECTED_SOURCE_LOCK_SHA256,

    "stage26_7b_formula_lock_sha256":
        EXPECTED_FORMULA_LOCK_SHA256,

    "stage26_7b_boundary_lock_sha256":
        EXPECTED_BOUNDARY_LOCK_SHA256,

    "batch_scaling_csv_sha256":
        batch_sha,

    "two_core_scaling_csv_sha256":
        two_core_sha,

    "group_B_component_capacity_csv_sha256":
        component_sha,

    "raw_extraction_descriptive_sha256":
        raw_sha,

    "results_json_sha256":
        results_sha,

    "receipt_sha256":
        receipt_sha,

    "file_count_excluding_manifest":
        len(
            manifest_rows
        ),

    "files":
        manifest_rows,

    "scientific_state": {
        "descriptive_capacity_results_computed":
            True,

        "new_measurement_performed":
            False,

        "complete_E2E_reconstructed":
            False,

        "pipeline_bottleneck_declared":
            False,

        "Pareto_changed":
            False,

        "GPU_used":
            False,
    },
}


atomic_json(
    MANIFEST,
    manifest_payload,
)


manifest_sha = sha256_file(
    MANIFEST
)


# =============================================================================
# 18. RESULT SUMMARY
# =============================================================================

banner(
    "STAGE26-7C :: RESULT SUMMARY"
)


print(
    "Batch-scaling numeric PASS rows:",
    len(
        batch_numeric_rows
    )
)

print(
    "CPU2/CPU1 matched PASS rows     :",
    len(
        two_core_rows
    )
)

print(
    "Group-B component ratios        :",
    len(
        component_rows
    )
)


print(
    "\nGROUP-B COMPONENT CAPACITY:"
)


for row in component_rows:

    print(
        f"  {row['target_id']:34s} "
        f"B={row['batch_size']:4d} "
        f"ratio={row['component_capacity_ratio_inference_over_representation']:.6f} "
        f"{row['lower_measured_capacity_component']}"
    )


print(
    "\nRAW EXTRACTION — DESCRIPTIVE ONLY:"
)


for key, value in raw_descriptive[
    "median_component_metrics"
].items():

    print(
        f"  {key:56s} {value}"
    )


print(
    "\nPROHIBITED RESULTS:"
)

print(
    "  complete E2E reconstructed : NO"
)

print(
    "  pipeline bottleneck        : NO"
)

print(
    "  missing cost imputed       : NO"
)

print(
    "  Pareto recomputed          : NO"
)

print(
    "  cross-group Pareto         : NO"
)


# =============================================================================
# 19. LOCAL PACKAGE AUDIT
# =============================================================================

banner(
    "STAGE26-7C :: LOCAL PACKAGE AUDIT"
)


for row in manifest_rows:

    path = (
        REPO
        /
        row[
            "repo_relative_path"
        ]
    )

    size = int(
        path.stat().st_size
    )

    digest = sha256_file(
        path
    )

    passed = (
        size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        digest
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{size:10,d} B "
        f"{digest} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Local Stage26-7C package audit failed."
        )


print(
    "\nManifest SHA256:"
)

print(
    " ",
    manifest_sha
)


# =============================================================================
# 20. GIT CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-7C :: GIT CHANGE AUDIT"
)


repo_status = git(
    "status",
    "--porcelain",
)


print(
    repo_status
)


if not repo_status:

    raise RuntimeError(
        "Expected uncommitted Stage26-7C package."
    )


unexpected = []


for line in repo_status.splitlines():

    relpath = line[
        3:
    ]


    if not relpath.startswith(
        str(
            CHECKPOINT_REL
        )
        + "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository changes:\n"
        + "\n".join(
            unexpected
        )
    )


# =============================================================================
# 21. GIT IDENTITY
# =============================================================================

author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_PARENT,
)

author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_PARENT,
)


if (
    not author_name.strip()
    or
    "@"
    not in
    author_email
):

    raise RuntimeError(
        "Could not recover Git identity."
    )


git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


print(
    "\nGit author:"
)

print(
    " ",
    author_name,
    "<" + author_email + ">"
)


# =============================================================================
# 22. COMMIT
# =============================================================================

banner(
    "STAGE26-7C :: COMMIT"
)


git(
    "add",
    str(
        CHECKPOINT_REL
    ),
)


print(
    git(
        "diff",
        "--cached",
        "--name-status",
    )
)


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-7C commit parent mismatch."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Stage26-7C commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository not clean after Stage26-7C commit."
    )


# =============================================================================
# 23. PUSH
# =============================================================================

banner(
    "STAGE26-7C :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    push_result = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
        text=True,
    )


    print(
        push_result.stdout.strip()
    )


github_token = None


# =============================================================================
# 24. REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-7C :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "origin/main did not advance to Stage26-7C."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote Stage26-7C parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote Stage26-7C subject mismatch."
    )


# =============================================================================
# 25. REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-7C :: REMOTE BYTE VERIFICATION"
)


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    str(
        MANIFEST.relative_to(
            REPO
        )
    ),
)


remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "Remote manifest SHA256:"
)

print(
    " ",
    remote_manifest_sha
)


if remote_manifest_sha != manifest_sha:

    raise RuntimeError(
        "Remote Stage26-7C manifest mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


for row in remote_manifest[
    "files"
]:

    data = git_blob_bytes(
        "origin/main",
        row[
            "repo_relative_path"
        ],
    )

    actual_size = len(
        data
    )

    actual_sha = sha256_bytes(
        data
    )

    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Remote Stage26-7C byte verification failed."
        )


# =============================================================================
# 26. REMOTE SCIENTIFIC VERIFICATION
# =============================================================================

banner(
    "STAGE26-7C :: REMOTE SCIENTIFIC VERIFICATION"
)


remote_results = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            RESULTS_JSON.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_receipt = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            RECEIPT.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)


remote_checks = {
    "result PASS":
        (
            remote_results[
                "status"
            ]
            ==
            "PASS_FROZEN_DESCRIPTIVE_CPU_CAPACITY_AND_SCALING"
        ),

    "73 PASS scaling rows":
        (
            remote_results[
                "batch_scaling"
            ][
                "PASS_numeric_rows"
            ]
            ==
            73
        ),

    "seven Group-B ratios":
        (
            remote_results[
                "group_B_component_capacity"
            ][
                "ratio_count"
            ]
            ==
            7
        ),

    "component-only claim":
        (
            remote_results[
                "group_B_component_capacity"
            ][
                "claim_boundary"
            ]
            ==
            (
                "DOWNSTREAM_COMPONENT_LEVEL_ONLY__"
                "NOT_COMPLETE_E2E__NOT_RAW_PCAP_TO_MODEL"
            )
        ),

    "4C3 caveat retained":
        (
            remote_results[
                "group_B_component_capacity"
            ][
                "representation_measurement_caveat"
            ]
            ==
            "OPEN_STAGE26_8_AUDIT_ITEM"
        ),

    "raw extraction not used in ratio":
        (
            remote_receipt[
                "scientific_state"
            ][
                "raw_extraction_used_in_ratio"
            ]
            is False
        ),

    "complete E2E false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "complete_E2E_reconstructed"
            ]
            is False
        ),

    "missing imputation false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "missing_cost_imputed"
            ]
            is False
        ),

    "pipeline bottleneck false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "full_pipeline_bottleneck_declared"
            ]
            is False
        ),

    "ratio uncertainty false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "ratio_uncertainty_computed"
            ]
            is False
        ),

    "new bootstrap false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "new_bootstrap_executed"
            ]
            is False
        ),

    "Pareto recomputed false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "Pareto_recomputed"
            ]
            is False
        ),

    "Pareto changed false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "Pareto_changed"
            ]
            is False
        ),

    "inference false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "inference_performed"
            ]
            is False
        ),

    "timing false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "timing_performed"
            ]
            is False
        ),

    "GPU false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "GPU_used"
            ]
            is False
        ),
}


for name, passed in remote_checks.items():

    print(
        f"{name:40s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    remote_checks.values()
):

    raise RuntimeError(
        "Remote Stage26-7C scientific verification failed."
    )


# =============================================================================
# 27. FINAL CLOSURE
# =============================================================================

banner(
    "STAGE26-7C CPU CAPACITY / SCALING COMPLETE"
)


final_status = git(
    "status",
    "--porcelain",
)


print(
    "NEW DURABLE COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nPARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nRESULT COUNTS:"
)

print(
    "  batch-scaling conditions       :",
    len(
        batch_scaling_rows
    )
)

print(
    "  numerical PASS scaling rows    :",
    len(
        batch_numeric_rows
    )
)

print(
    "  CPU2/CPU1 matched PASS rows    :",
    len(
        two_core_rows
    )
)

print(
    "  Group-B component ratios       :",
    len(
        component_rows
    )
)


print(
    "\nGROUP-B COMPONENT CAPACITY:"
)


for row in component_rows:

    print(
        f"  {row['target_id']:34s} "
        f"B={row['batch_size']:4d} "
        f"ratio={row['component_capacity_ratio_inference_over_representation']:.6f} "
        f"{row['lower_measured_capacity_component']}"
    )


print(
    "\nRAW EXTRACTION:"
)

print(
    "  reported descriptively only : YES"
)

print(
    "  used in pipeline ratio      : NO"
)


print(
    "\nSCIENTIFIC BOUNDARIES:"
)

print(
    "  complete E2E reconstructed      : NO"
)

print(
    "  missing cost imputed            : NO"
)

print(
    "  full-pipeline bottleneck         : NO"
)

print(
    "  ratio uncertainty propagated    : NO"
)

print(
    "  new bootstrap                   : NO"
)

print(
    "  Pareto recomputed               : NO"
)

print(
    "  Pareto changed                  : NO"
)

print(
    "  new inference                   : NO"
)

print(
    "  new timing                      : NO"
)

print(
    "  GPU                             : NO"
)


print(
    "\n4C3 CAVEAT:"
)

print(
    "  OPEN_STAGE26_8_AUDIT_ITEM"
)


print(
    "\nHASHES:"
)

print(
    "  batch scaling CSV :",
    batch_sha
)

print(
    "  two-core CSV      :",
    two_core_sha
)

print(
    "  Group-B CSV       :",
    component_sha
)

print(
    "  raw descriptive   :",
    raw_sha
)

print(
    "  results JSON      :",
    results_sha
)

print(
    "  receipt           :",
    receipt_sha
)

print(
    "  manifest          :",
    manifest_sha
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  commit             : PASS"
)

print(
    "  parent             : PASS"
)

print(
    "  package bytes      : PASS"
)

print(
    "  scaling semantics  : PASS"
)

print(
    "  boundary semantics : PASS"
)

print(
    "  scientific content : PASS"
)


print(
    "\nRepo clean:",
    final_status == ""
)


if final_status:

    raise RuntimeError(
        "Repository not clean after Stage26-7C."
    )


print(
    "\nNEXT:"
)

print(
    "  Stage26-8 CPU closure and publication audit."
)

print(
    "  The first required Stage26-8 task is the explicit"
)

print(
    "  Stage26-4C3 inter-iteration integrity-read sensitivity audit."
)

print(
    "  GPU remains OFF until Stage26-8 is completed, committed,"
)

print(
    "  pushed, and remotely verified."
)


STAGE26-7C :: DURABLE SCIENTIFIC PARENT
Expected parent: c524e763b8ae7593204f95f23f9e819b4dcaaf9f
Local HEAD     : c524e763b8ae7593204f95f23f9e819b4dcaaf9f
origin/main    : c524e763b8ae7593204f95f23f9e819b4dcaaf9f
Repo clean     : True

STAGE26-7C :: EXACT STAGE26-7B LOCK IDENTITY
source lock              PASS c56a08cc88a2e8d75df0832b72e41ecbb90ee1cf3f12c0a7402d9fc1d5247c42
formula lock             PASS 7cb33c57dd030e6fb2c0763eec70471b4afc0f9d06d071bc690fbac02d6dacea
boundary lock            PASS 1def25c4046770dd1da2ff7939c38b01bc19917bbecf01423c9cad684b68221c
freeze receipt           PASS 0bf2b935e8b5509b4fa6d32413a9ce835af7ead4166a77f173f553f940369790
lock manifest            PASS f3f844c1fb3953831cd345f5ac767730e0933efb068ad9b7a79e50ccfe5789e8

STAGE26-7C :: LOCKED FORMULA / BOUNDARY SEMANTICS
Batch scaling                 : ALLOWED
CPU2 vs CPU1 scaling          : ALLOWED
Group-B component ratio       : ALLOWED — component level only
Raw extraction pipeline ratio : PROHIBITED
Compl

In [3]:
# =============================================================================
# STAGE26-8A
# READ-ONLY STAGE26-4C3 INTER-ITERATION INTEGRITY-READ IMPLEMENTATION AUDIT
#
# DURABLE SCIENTIFIC PARENT:
#   9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3
#
# PURPOSE
# -------
# Resolve the known Stage26-4C3 methodology caveat BEFORE final CPU closure.
#
# KNOWN OPEN QUESTION
# -------------------
# Stage26-4C3 representation timing may perform a full-array output
# integrity/fingerprint read OUTSIDE each timer window, between timed
# iterations.
#
# Even though such a read is not included in the measured latency itself,
# it may alter cache / CPU state entering the next timed iteration.
#
# THIS CELL DOES NOT RERUN 4C3.
#
# It determines, from the exact committed implementation:
#
#   1. which worker/source actually produced Stage26-4C3;
#   2. exact worker SHA256;
#   3. timer-start / timer-stop locations;
#   4. fingerprint/hash/output-read locations;
#   5. ordering relative to repeated timed iterations;
#   6. whether the integrity read is:
#
#        INSIDE_TIMER
#        OUTSIDE_TIMER_BEFORE_NEXT_TIMED_ITERATION
#        OUTSIDE_TIMER_AFTER_ALL_TIMED_ITERATIONS
#        NOT_RESOLVABLE_FROM_SOURCE
#
#   7. whether a prospective sensitivity run is scientifically justified.
#
# IMPORTANT
# ---------
# No original Stage26-4C3 artifact is modified.
# No replacement result is calculated.
# No sensitivity timing is run yet.
#
# NO:
#   - model loading
#   - inference
#   - timing execution
#   - bootstrap
#   - ratio calculation
#   - Pareto recomputation
#   - PCAP
#   - Release corpus
#   - GPU
#   - Git modification
#
# OUTPUT:
#   TRANSIENT ONLY
#
#   /kaggle/working/stage26_deployment_profiling/cpu_closure/
#       stage26_8a_4c3_integrity_read_implementation_audit.json
# =============================================================================

from __future__ import annotations

import ast
import json
import re
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3"
)

RUNTIME_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

CPU_CLOSURE_RUNTIME = (
    RUNTIME_ROOT
    / "cpu_closure"
)

OUT = (
    CPU_CLOSURE_RUNTIME
    / "stage26_8a_4c3_integrity_read_implementation_audit.json"
)


# -----------------------------------------------------------------------------
# Stage26-7C exact durable identities
# -----------------------------------------------------------------------------

STAGE26_7C_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_7c_cpu_capacity_scaling"
)

STAGE26_7C_RESULTS = (
    STAGE26_7C_DIR
    / "stage26_7c_capacity_scaling_results.json"
)

STAGE26_7C_RECEIPT = (
    STAGE26_7C_DIR
    / "stage26_7c_capacity_scaling_receipt.json"
)

STAGE26_7C_MANIFEST = (
    STAGE26_7C_DIR
    / "stage26_7c_capacity_scaling_manifest.json"
)


EXPECTED_7C_RESULTS_SHA256 = (
    "ee5a852fb43ce01c980e64ef83611a88808d3a11a56a078609467fb899e39fdf"
)

EXPECTED_7C_RECEIPT_SHA256 = (
    "30bfd4657a5a0218ce22a4fab1922ee26e07496c148809ec04c8059656b73ce4"
)

EXPECTED_7C_MANIFEST_SHA256 = (
    "f090a992195e718b9ae569e681b3fd4b50b9dba76bca89fb6fc1099e30fa426c"
)


# -----------------------------------------------------------------------------
# Stage26-4C3 durable measurement artifacts
# -----------------------------------------------------------------------------

REP_TIMING_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c3_cpu1_representation_timing"
)

REP_SUMMARY_CSV = (
    REP_TIMING_DIR
    / "stage26_4c3_cpu1_representation_summary.csv"
)

REP_SUMMARY_JSON = (
    REP_TIMING_DIR
    / "stage26_4c3_cpu1_representation_summary.json"
)

REP_RECEIPT = (
    REP_TIMING_DIR
    / "stage26_4c3_cpu1_representation_timing_receipt.json"
)

REP_MANIFEST = (
    REP_TIMING_DIR
    / "stage26_4c3_cpu1_representation_timing_manifest.json"
)

REP_PLAN = (
    REP_TIMING_DIR
    / "stage26_4c3_execution_plan.json"
)

REP_RAW = (
    REP_TIMING_DIR
    / "stage26_4c3_raw_observations.csv"
)


EXPECTED_REP_SUMMARY_CSV_SHA256 = (
    "ca5bbc17edfa4bb088599236b4099c1986fd73cb7f087ef3ac753a524afd2ba5"
)

EXPECTED_REP_SUMMARY_JSON_SHA256 = (
    "63c8041144668c62218fc9c46eb6a57aba9dcda0c9a3f0477f932f6a572c34f7"
)

EXPECTED_REP_RECEIPT_SHA256 = (
    "ed55b65905080dd6b796a54fb720de76bf100fdc7f3271a973e49c81f1722b14"
)

EXPECTED_REP_MANIFEST_SHA256 = (
    "bfbc9000a708a803c53a11b591e8021d54325d1b7bccebb4b0aad65e1439b9db"
)

EXPECTED_REP_PLAN_SHA256 = (
    "5e23b40c0e5d13a91641c42ad3ac95b4f3a828a6e351a3cd139d826d3ade2568"
)

EXPECTED_REP_RAW_SHA256 = (
    "189852745ef83330526f6c680d459c7075a695f97614aa6c76a7ae7ded3447b3"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            p.stdout
        )

    return p


def git(*args):

    return run(
        [
            "git",
            *args,
        ]
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def read_json(path):

    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def context(
    lines,
    line_number,
    radius=10,
):

    start = max(
        1,
        line_number - radius,
    )

    end = min(
        len(lines),
        line_number + radius,
    )


    return "\n".join(
        f"{n:6d}: {lines[n - 1]}"
        for n in range(
            start,
            end + 1,
        )
    )


def recursive_find_strings(
    obj,
    *,
    path="$",
    result=None,
):

    if result is None:

        result = []


    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            child = (
                path
                +
                "."
                +
                str(key)
            )

            recursive_find_strings(
                value,
                path=child,
                result=result,
            )


    elif isinstance(
        obj,
        list,
    ):

        for index, value in enumerate(obj):

            recursive_find_strings(
                value,
                path=f"{path}[{index}]",
                result=result,
            )


    elif isinstance(
        obj,
        str,
    ):

        result.append(
            {
                "path":
                    path,

                "value":
                    obj,
            }
        )


    return result


def repo_file_candidates():

    tracked = git(
        "ls-files"
    ).splitlines()


    result = []


    for relpath in tracked:

        lower = relpath.lower()

        suffix = Path(
            relpath
        ).suffix.lower()


        if suffix != ".py":

            continue


        if (
            "stage26"
            not in lower
            and
            "representation"
            not in lower
            and
            "packet_image"
            not in lower
        ):

            continue


        result.append(
            relpath
        )


    return result


# =============================================================================
# 2. DURABLE STATE
# =============================================================================

banner(
    "STAGE26-8A :: DURABLE SCIENTIFIC STATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)

print(
    "Transient output exists:",
    OUT.exists()
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-8A parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Stage26-8A."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if OUT.exists():

    raise RuntimeError(
        "Stage26-8A transient audit already exists."
    )


# =============================================================================
# 3. STAGE26-7C CLOSURE IDENTITY
# =============================================================================

banner(
    "STAGE26-8A :: STAGE26-7C ANCHOR IDENTITY"
)


anchor_checks = [
    (
        "7C results",
        STAGE26_7C_RESULTS,
        EXPECTED_7C_RESULTS_SHA256,
    ),

    (
        "7C receipt",
        STAGE26_7C_RECEIPT,
        EXPECTED_7C_RECEIPT_SHA256,
    ),

    (
        "7C manifest",
        STAGE26_7C_MANIFEST,
        EXPECTED_7C_MANIFEST_SHA256,
    ),
]


for label, path, expected in anchor_checks:

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:20s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Stage26-7C identity mismatch: {label}"
        )


# =============================================================================
# 4. EXACT 4C3 ARTIFACT IDENTITY
# =============================================================================

banner(
    "STAGE26-8A :: EXACT STAGE26-4C3 ARTIFACT IDENTITY"
)


rep_checks = [
    (
        "summary CSV",
        REP_SUMMARY_CSV,
        EXPECTED_REP_SUMMARY_CSV_SHA256,
    ),

    (
        "summary JSON",
        REP_SUMMARY_JSON,
        EXPECTED_REP_SUMMARY_JSON_SHA256,
    ),

    (
        "receipt",
        REP_RECEIPT,
        EXPECTED_REP_RECEIPT_SHA256,
    ),

    (
        "manifest",
        REP_MANIFEST,
        EXPECTED_REP_MANIFEST_SHA256,
    ),

    (
        "execution plan",
        REP_PLAN,
        EXPECTED_REP_PLAN_SHA256,
    ),

    (
        "raw observations",
        REP_RAW,
        EXPECTED_REP_RAW_SHA256,
    ),
]


rep_identity = {}


for label, path, expected in rep_checks:

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:22s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Stage26-4C3 identity mismatch: {label}"
        )


    rep_identity[
        label
    ] = {
        "repo_relative_path":
            str(
                path.relative_to(
                    REPO
                )
            ),

        "sha256":
            actual,
    }


# =============================================================================
# 5. SEARCH 4C3 METADATA FOR WORKER/SOURCE REFERENCES
# =============================================================================

banner(
    "STAGE26-8A :: WORKER / IMPLEMENTATION REFERENCE DISCOVERY"
)


metadata_objects = {
    "receipt":
        read_json(
            REP_RECEIPT
        ),

    "manifest":
        read_json(
            REP_MANIFEST
        ),

    "execution_plan":
        read_json(
            REP_PLAN
        ),

    "summary":
        read_json(
            REP_SUMMARY_JSON
        ),
}


metadata_string_hits = []


for source_name, obj in metadata_objects.items():

    strings = recursive_find_strings(
        obj
    )


    for item in strings:

        value_lower = item[
            "value"
        ].lower()


        if (
            ".py"
            in value_lower
            or
            "worker"
            in value_lower
            or
            "representation"
            in value_lower
            or
            "fingerprint"
            in value_lower
            or
            "sha256"
            in item[
                "path"
            ].lower()
        ):

            metadata_string_hits.append(
                {
                    "metadata_source":
                        source_name,

                    **item,
                }
            )


print(
    "Relevant metadata string references:",
    len(
        metadata_string_hits
    )
)


for item in metadata_string_hits[
    :120
]:

    print(
        f"\n[{item['metadata_source']}] "
        f"{item['path']} = {item['value']!r}"
    )


# =============================================================================
# 6. DISCOVER CANDIDATE PYTHON IMPLEMENTATIONS
# =============================================================================

banner(
    "STAGE26-8A :: CANDIDATE IMPLEMENTATION FILES"
)


candidates = repo_file_candidates()


candidate_records = []


strong_terms = [
    "perf_counter_ns",
    "time_ns",
    "fingerprint",
    "sha256",
    "representation",
    "materialize",
    "timed_runs",
    "warmup_runs",
]


for relpath in candidates:

    path = (
        REPO
        /
        relpath
    )


    try:

        text = path.read_text(
            encoding="utf-8",
            errors="replace",
        )

    except Exception:

        continue


    lower = text.lower()


    hit_terms = [
        term
        for term in strong_terms
        if term.lower()
        in
        lower
    ]


    if not hit_terms:

        continue


    record = {
        "repo_relative_path":
            relpath,

        "sha256":
            sha256_file(
                path
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "hit_terms":
            hit_terms,
    }


    candidate_records.append(
        record
    )


candidate_records.sort(
    key=lambda x: (
        -len(
            x[
                "hit_terms"
            ]
        ),
        x[
            "repo_relative_path"
        ],
    )
)


print(
    "Candidate count:",
    len(
        candidate_records
    )
)


for index, record in enumerate(
    candidate_records[
        :80
    ],
    start=1,
):

    print(
        f"\n[{index}] "
        f"{record['sha256']} "
        f"{record['repo_relative_path']}"
    )

    print(
        "    hits:",
        record[
            "hit_terms"
        ]
    )


# =============================================================================
# 7. SCORE CANDIDATES FOR 4C3 WORKER LIKELIHOOD
# =============================================================================

banner(
    "STAGE26-8A :: WORKER RESOLUTION"
)


metadata_text = json.dumps(
    metadata_objects,
    sort_keys=True,
)


scored = []


for record in candidate_records:

    relpath = record[
        "repo_relative_path"
    ]

    basename = Path(
        relpath
    ).name


    score = 0

    reasons = []


    if relpath in metadata_text:

        score += 100

        reasons.append(
            "full path referenced in 4C3 metadata"
        )


    if basename in metadata_text:

        score += 50

        reasons.append(
            "basename referenced in 4C3 metadata"
        )


    lower = relpath.lower()


    if "stage26" in lower:

        score += 10

        reasons.append(
            "Stage26 path"
        )


    if "representation" in lower:

        score += 15

        reasons.append(
            "representation path"
        )


    if "4c" in lower:

        score += 20

        reasons.append(
            "4C path"
        )


    if "worker" in lower:

        score += 10

        reasons.append(
            "worker filename/path"
        )


    if "perf_counter_ns" in record[
        "hit_terms"
    ]:

        score += 8

        reasons.append(
            "contains perf_counter_ns"
        )


    if "fingerprint" in record[
        "hit_terms"
    ]:

        score += 8

        reasons.append(
            "contains fingerprint"
        )


    if "timed_runs" in record[
        "hit_terms"
    ]:

        score += 6

        reasons.append(
            "contains timed_runs"
        )


    scored.append(
        {
            **record,

            "score":
                score,

            "reasons":
                reasons,
        }
    )


scored.sort(
    key=lambda x: (
        -x[
            "score"
        ],
        x[
            "repo_relative_path"
        ],
    )
)


for index, record in enumerate(
    scored[
        :20
    ],
    start=1,
):

    print(
        f"\n[{index}] score={record['score']} "
        f"{record['repo_relative_path']}"
    )

    print(
        "    SHA256:",
        record[
            "sha256"
        ]
    )

    print(
        "    reasons:",
        record[
            "reasons"
        ]
    )


if not scored:

    raise RuntimeError(
        "No candidate Stage26-4C3 implementation source discovered."
    )


top_score = scored[
    0
][
    "score"
]


top_candidates = [
    record
    for record in scored
    if record[
        "score"
    ]
    ==
    top_score
]


print(
    "\nTop-score candidate count:",
    len(
        top_candidates
    )
)


if len(
    top_candidates
) != 1:

    print(
        "\nWorker resolution is ambiguous."
    )

    worker_record = None

else:

    worker_record = top_candidates[
        0
    ]


    print(
        "\nResolved worker candidate:"
    )

    print(
        " ",
        worker_record[
            "repo_relative_path"
        ]
    )

    print(
        " ",
        worker_record[
            "sha256"
        ]
    )


# =============================================================================
# 8. SOURCE-LEVEL TIMER / FINGERPRINT AUDIT
# =============================================================================

banner(
    "STAGE26-8A :: TIMER / INTEGRITY-READ SOURCE AUDIT"
)


source_analysis = {
    "resolved":
        False,

    "timer_lines":
        [],

    "fingerprint_lines":
        [],

    "hash_lines":
        [],

    "loop_lines":
        [],

    "contexts":
        [],
}


classification = (
    "NOT_RESOLVABLE_FROM_SOURCE"
)


if worker_record is not None:

    worker_path = (
        REPO
        /
        worker_record[
            "repo_relative_path"
        ]
    )


    source_text = worker_path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    source_lines = source_text.splitlines()


    timer_regex = re.compile(
        r"(?i)"
        r"(perf_counter_ns"
        r"|perf_counter"
        r"|monotonic_ns"
        r"|time_ns)"
    )


    fingerprint_regex = re.compile(
        r"(?i)"
        r"(fingerprint"
        r"|sha256"
        r"|hashlib"
        r"|tobytes"
        r"|digest)"
    )


    loop_regex = re.compile(
        r"^\s*(for|while)\s+"
    )


    timer_lines = [
        i
        for i, line in enumerate(
            source_lines,
            start=1,
        )
        if timer_regex.search(
            line
        )
    ]


    fingerprint_lines = [
        i
        for i, line in enumerate(
            source_lines,
            start=1,
        )
        if fingerprint_regex.search(
            line
        )
    ]


    loop_lines = [
        i
        for i, line in enumerate(
            source_lines,
            start=1,
        )
        if loop_regex.search(
            line
        )
    ]


    source_analysis[
        "resolved"
    ] = True

    source_analysis[
        "timer_lines"
    ] = timer_lines

    source_analysis[
        "fingerprint_lines"
    ] = fingerprint_lines

    source_analysis[
        "hash_lines"
    ] = fingerprint_lines

    source_analysis[
        "loop_lines"
    ] = loop_lines


    print(
        "Timer-related lines:",
        timer_lines
    )

    print(
        "Fingerprint/hash-related lines:",
        fingerprint_lines
    )

    print(
        "Loop lines:",
        loop_lines
    )


    interesting_lines = sorted(
        set(
            timer_lines
            +
            fingerprint_lines
            +
            loop_lines
        )
    )


    for line_number in interesting_lines:

        ctx = context(
            source_lines,
            line_number,
            radius=8,
        )


        source_analysis[
            "contexts"
        ].append(
            {
                "line":
                    line_number,

                "context":
                    ctx,
            }
        )


        print(
            "\n"
            +
            "-" * 124
        )

        print(
            f"Context near line {line_number}"
        )

        print(
            "-" * 124
        )

        print(
            ctx
        )


# =============================================================================
# 9. AST-BASED STRUCTURAL AUDIT
# =============================================================================

banner(
    "STAGE26-8A :: AST STRUCTURAL AUDIT"
)


ast_evidence = []


if worker_record is not None:

    try:

        tree = ast.parse(
            source_text
        )

    except SyntaxError as exc:

        print(
            "AST parse failed:",
            repr(
                exc
            )
        )

        tree = None


    if tree is not None:

        parents = {}


        for parent in ast.walk(
            tree
        ):

            for child in ast.iter_child_nodes(
                parent
            ):

                parents[
                    child
                ] = parent


        for node in ast.walk(
            tree
        ):

            if not isinstance(
                node,
                ast.Call,
            ):

                continue


            try:

                call_text = ast.unparse(
                    node.func
                )

            except Exception:

                call_text = ""


            call_lower = call_text.lower()


            if not any(
                token
                in
                call_lower
                for token in [
                    "perf_counter",
                    "sha256",
                    "fingerprint",
                    "tobytes",
                    "digest",
                ]
            ):

                continue


            # Find enclosing loop/function.
            parent = parents.get(
                node
            )

            enclosing_loop = None

            enclosing_function = None


            while parent is not None:

                if (
                    enclosing_loop is None
                    and
                    isinstance(
                        parent,
                        (
                            ast.For,
                            ast.While,
                        ),
                    )
                ):

                    enclosing_loop = {
                        "type":
                            type(
                                parent
                            ).__name__,

                        "lineno":
                            getattr(
                                parent,
                                "lineno",
                                None,
                            ),

                        "end_lineno":
                            getattr(
                                parent,
                                "end_lineno",
                                None,
                            ),
                    }


                if (
                    enclosing_function is None
                    and
                    isinstance(
                        parent,
                        (
                            ast.FunctionDef,
                            ast.AsyncFunctionDef,
                        ),
                    )
                ):

                    enclosing_function = {
                        "name":
                            parent.name,

                        "lineno":
                            getattr(
                                parent,
                                "lineno",
                                None,
                            ),

                        "end_lineno":
                            getattr(
                                parent,
                                "end_lineno",
                                None,
                            ),
                    }


                parent = parents.get(
                    parent
                )


            record = {
                "call":
                    call_text,

                "lineno":
                    getattr(
                        node,
                        "lineno",
                        None,
                    ),

                "end_lineno":
                    getattr(
                        node,
                        "end_lineno",
                        None,
                    ),

                "enclosing_loop":
                    enclosing_loop,

                "enclosing_function":
                    enclosing_function,
            }


            ast_evidence.append(
                record
            )


        for item in ast_evidence:

            print(
                "\n",
                json.dumps(
                    item,
                    indent=2,
                    sort_keys=True,
                ),
                sep="",
            )


# =============================================================================
# 10. ORDERING CLASSIFICATION
# =============================================================================

banner(
    "STAGE26-8A :: INTEGRITY-READ ORDERING CLASSIFICATION"
)


ordering_evidence = []


if worker_record is not None:

    # -------------------------------------------------------------------------
    # Conservative line-order analysis.
    #
    # We do NOT claim semantic certainty solely from regex.
    # We classify only if a repeated loop visibly contains:
    #
    #   timer start
    #   measured work
    #   timer end
    #   fingerprint/hash read
    #
    # before the loop body ends.
    # -------------------------------------------------------------------------

    try:

        tree_for_loops = ast.parse(
            source_text
        )

    except SyntaxError:

        tree_for_loops = None


    if tree_for_loops is not None:

        for node in ast.walk(
            tree_for_loops
        ):

            if not isinstance(
                node,
                (
                    ast.For,
                    ast.While,
                ),
            ):

                continue


            loop_start = getattr(
                node,
                "lineno",
                None,
            )

            loop_end = getattr(
                node,
                "end_lineno",
                None,
            )


            if (
                loop_start is None
                or
                loop_end is None
            ):

                continue


            loop_timer_lines = [
                line
                for line in source_analysis[
                    "timer_lines"
                ]
                if loop_start
                <=
                line
                <=
                loop_end
            ]


            loop_fingerprint_lines = [
                line
                for line in source_analysis[
                    "fingerprint_lines"
                ]
                if loop_start
                <=
                line
                <=
                loop_end
            ]


            if not loop_timer_lines:

                continue


            record = {
                "loop_type":
                    type(
                        node
                    ).__name__,

                "loop_start":
                    loop_start,

                "loop_end":
                    loop_end,

                "timer_lines":
                    loop_timer_lines,

                "fingerprint_lines":
                    loop_fingerprint_lines,
            }


            ordering_evidence.append(
                record
            )


        for item in ordering_evidence:

            print(
                json.dumps(
                    item,
                    indent=2,
                    sort_keys=True,
                )
            )


    # -------------------------------------------------------------------------
    # Classification.
    # -------------------------------------------------------------------------

    evidence_inside_repeated_loop = []


    for item in ordering_evidence:

        timers = sorted(
            item[
                "timer_lines"
            ]
        )

        fps = sorted(
            item[
                "fingerprint_lines"
            ]
        )


        if len(
            timers
        ) < 2:

            continue


        timer_start = timers[
            0
        ]

        timer_end = timers[
            -1
        ]


        fps_after_timer_end = [
            line
            for line in fps
            if line
            >
            timer_end
        ]


        fps_between_timers = [
            line
            for line in fps
            if timer_start
            <=
            line
            <=
            timer_end
        ]


        if fps_between_timers:

            evidence_inside_repeated_loop.append(
                (
                    "INSIDE_TIMER",
                    item,
                    fps_between_timers,
                )
            )


        if fps_after_timer_end:

            evidence_inside_repeated_loop.append(
                (
                    "OUTSIDE_TIMER_BEFORE_NEXT_TIMED_ITERATION",
                    item,
                    fps_after_timer_end,
                )
            )


    labels = [
        item[
            0
        ]
        for item in evidence_inside_repeated_loop
    ]


    if (
        "OUTSIDE_TIMER_BEFORE_NEXT_TIMED_ITERATION"
        in
        labels
    ):

        classification = (
            "OUTSIDE_TIMER_BEFORE_NEXT_TIMED_ITERATION"
        )


    elif "INSIDE_TIMER" in labels:

        classification = (
            "INSIDE_TIMER"
        )


    else:

        # Check fingerprint calls that occur after all timer loops.
        all_timer_lines = source_analysis[
            "timer_lines"
        ]

        all_fp_lines = source_analysis[
            "fingerprint_lines"
        ]


        if (
            all_timer_lines
            and
            all_fp_lines
            and
            min(
                all_fp_lines
            )
            >
            max(
                all_timer_lines
            )
        ):

            classification = (
                "OUTSIDE_TIMER_AFTER_ALL_TIMED_ITERATIONS"
            )


        else:

            classification = (
                "NOT_RESOLVABLE_FROM_SOURCE"
            )


print(
    "Classification:"
)

print(
    " ",
    classification
)


# =============================================================================
# 11. SCIENTIFIC CONSEQUENCE
# =============================================================================

banner(
    "STAGE26-8A :: SCIENTIFIC CONSEQUENCE"
)


if classification == (
    "OUTSIDE_TIMER_BEFORE_NEXT_TIMED_ITERATION"
):

    consequence = (
        "The Stage26-4C3 integrity/fingerprint read is excluded from the "
        "reported timer window but occurs between repeated timed iterations. "
        "Therefore a cache/CPU-state sensitivity concern is real. Preserve "
        "Stage26-4C3 as the original durable measurement and prospectively "
        "freeze a cleaner sensitivity implementation before any sensitivity "
        "timing is executed."
    )

    recommended_next = (
        "STAGE26-8B_FREEZE_PROSPECTIVE_4C3_SENSITIVITY_IMPLEMENTATION"
    )


elif classification == (
    "INSIDE_TIMER"
):

    consequence = (
        "The integrity/fingerprint work appears inside the timing window. "
        "This is a stronger measurement-boundary issue than previously "
        "documented and requires a prospective sensitivity protocol before "
        "publication closure."
    )

    recommended_next = (
        "STAGE26-8B_FREEZE_PROSPECTIVE_4C3_SENSITIVITY_IMPLEMENTATION"
    )


elif classification == (
    "OUTSIDE_TIMER_AFTER_ALL_TIMED_ITERATIONS"
):

    consequence = (
        "The integrity/fingerprint read occurs only after the repeated timed "
        "iterations, so the previously suspected inter-iteration cache-state "
        "contamination is not supported by the committed source. Stage26-4C3 "
        "can remain as measured, with the audit documenting why no sensitivity "
        "rerun is necessary."
    )

    recommended_next = (
        "STAGE26-8B_ANCHOR_NO_SENSITIVITY_RUN_REQUIRED"
    )


else:

    consequence = (
        "The committed source audit does not yet provide enough structural "
        "evidence to determine the exact integrity-read ordering. Perform one "
        "narrower provenance/source diagnostic before deciding whether any "
        "sensitivity run is justified."
    )

    recommended_next = (
        "NARROW_SOURCE_DIAGNOSTIC_REQUIRED"
    )


print(
    consequence
)

print(
    "\nRecommended next:"
)

print(
    " ",
    recommended_next
)


# =============================================================================
# 12. WRITE TRANSIENT AUDIT
# =============================================================================

banner(
    "STAGE26-8A :: WRITE TRANSIENT AUDIT"
)


CPU_CLOSURE_RUNTIME.mkdir(
    parents=True,
    exist_ok=True,
)


payload = {
    "schema":
        "stage26_8a_4c3_integrity_read_implementation_audit_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "status":
        "PASS_READ_ONLY_IMPLEMENTATION_AUDIT",

    "stage26_7c_anchor": {
        "commit":
            EXPECTED_PARENT,

        "results_sha256":
            EXPECTED_7C_RESULTS_SHA256,

        "receipt_sha256":
            EXPECTED_7C_RECEIPT_SHA256,

        "manifest_sha256":
            EXPECTED_7C_MANIFEST_SHA256,
    },

    "stage26_4c3_identity":
        rep_identity,

    "metadata_string_hits":
        metadata_string_hits,

    "candidate_implementation_files":
        scored,

    "resolved_worker":
        worker_record,

    "source_analysis":
        source_analysis,

    "ast_evidence":
        ast_evidence,

    "ordering_evidence":
        ordering_evidence,

    "classification":
        classification,

    "scientific_consequence":
        consequence,

    "recommended_next":
        recommended_next,

    "measurement_policy": {
        "original_stage26_4c3_retained":
            True,

        "original_stage26_4c3_overwritten":
            False,

        "sensitivity_result_computed":
            False,

        "sensitivity_timing_executed":
            False,

        "publication_replacement_decided":
            False,

        "stage26_7_component_ratios_recomputed":
            False,
    },

    "scientific_state": {
        "implementation_inspected":
            True,

        "timing_executed":
            False,

        "representation_reexecuted":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "bootstrap_executed":
            False,

        "Pareto_recomputed":
            False,

        "PCAP_accessed":
            False,

        "Release_corpus_accessed":
            False,

        "GPU_used":
            False,

        "Git_modified":
            False,
    },
}


with OUT.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        payload,
        f,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    )

    f.write(
        "\n"
    )


out_sha = sha256_file(
    OUT
)


print(
    "Output:"
)

print(
    " ",
    OUT
)

print(
    "SHA256:"
)

print(
    " ",
    out_sha
)


# =============================================================================
# 13. FINAL READ-ONLY CLOSURE
# =============================================================================

banner(
    "STAGE26-8A IMPLEMENTATION AUDIT COMPLETE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed during Stage26-8A."
    )


if final_remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed during Stage26-8A."
    )


if final_status:

    raise RuntimeError(
        "Stage26-8A unexpectedly modified Git."
    )


print(
    "\nFINAL CLASSIFICATION:"
)

print(
    " ",
    classification
)


print(
    "\nRECOMMENDED NEXT:"
)

print(
    " ",
    recommended_next
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  exact 4C3 artifacts verified       : YES"
)

print(
    "  committed implementation inspected : YES"
)

print(
    "  timer ordering inspected           : YES"
)

print(
    "  integrity-read ordering inspected  : YES"
)

print(
    "  original 4C3 modified              : NO"
)

print(
    "  sensitivity timing run             : NO"
)

print(
    "  new representation timing          : NO"
)

print(
    "  Stage26-7 ratios recomputed        : NO"
)

print(
    "  new bootstrap                      : NO"
)

print(
    "  Pareto recomputed                  : NO"
)

print(
    "  model loaded                       : NO"
)

print(
    "  inference                          : NO"
)

print(
    "  GPU                                : NO"
)

print(
    "  Git modified                       : NO"
)


STAGE26-8A :: DURABLE SCIENTIFIC STATE
Expected parent: 9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3
Local HEAD     : 9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3
origin/main    : 9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3
Repo clean     : True
Transient output exists: False

STAGE26-8A :: STAGE26-7C ANCHOR IDENTITY
7C results           PASS ee5a852fb43ce01c980e64ef83611a88808d3a11a56a078609467fb899e39fdf
7C receipt           PASS 30bfd4657a5a0218ce22a4fab1922ee26e07496c148809ec04c8059656b73ce4
7C manifest          PASS f090a992195e718b9ae569e681b3fd4b50b9dba76bca89fb6fc1099e30fa426c

STAGE26-8A :: EXACT STAGE26-4C3 ARTIFACT IDENTITY
summary CSV            PASS ca5bbc17edfa4bb088599236b4099c1986fd73cb7f087ef3ac753a524afd2ba5
summary JSON           PASS 63c8041144668c62218fc9c46eb6a57aba9dcda0c9a3f0477f932f6a572c34f7
receipt                PASS ed55b65905080dd6b796a54fb720de76bf100fdc7f3271a973e49c81f1722b14
manifest               PASS bfbc9000a708a803c53a11b591e8021d54325d1b7bccebb4b0aad65e1439

In [2]:
# =============================================================================
# STAGE26-8A-R0
# NARROW FRESH-RUNTIME GIT WORKSPACE RESTORE
#
# PURPOSE
# -------
# Restore the repository after Kaggle /kaggle/working was reset.
#
# EXPECTED DURABLE SCIENTIFIC ANCHOR:
#   9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3
#
# THIS CELL DOES NOT:
#   - run Stage26 science
#   - load models
#   - access datasets
#   - run inference
#   - run timing
#   - compute statistics
#   - modify scientific artifacts
#   - commit
#   - push
#   - use GPU
#
# It only:
#   1. clones the public GitHub repository if absent;
#   2. fetches origin/main;
#   3. verifies origin/main is the expected Stage26-7C anchor;
#   4. checks out that exact commit;
#   5. verifies a clean repository.
# =============================================================================

from pathlib import Path
import shutil
import subprocess


REPO_URL = (
    "https://github.com/themubasshir/"
    "ids2018-validation-safe-ablation.git"
)

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_ANCHOR = (
    "9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3"
)


def run(cmd, cwd=None, check=True):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    print(
        "$",
        " ".join(
            map(str, cmd)
        )
    )

    if p.stdout.strip():
        print(
            p.stdout.rstrip()
        )

    if check and p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return p


print(
    "=" * 100
)

print(
    "STAGE26-8A-R0 :: FRESH-RUNTIME WORKSPACE RESTORE"
)

print(
    "=" * 100
)


# -------------------------------------------------------------------------
# 1. Inspect current runtime
# -------------------------------------------------------------------------

print(
    "\n/kaggle/working exists:",
    Path(
        "/kaggle/working"
    ).is_dir()
)

print(
    "Repository exists:",
    REPO.exists()
)


# -------------------------------------------------------------------------
# 2. Clone only if repository is absent
# -------------------------------------------------------------------------

if not REPO.exists():

    run(
        [
            "git",
            "clone",
            "--branch",
            "main",
            "--single-branch",
            REPO_URL,
            str(REPO),
        ]
    )

else:

    if not (
        REPO
        /
        ".git"
    ).is_dir():

        raise RuntimeError(
            f"{REPO} exists but is not a Git repository. "
            "Do not delete it automatically."
        )


# -------------------------------------------------------------------------
# 3. Fetch exact remote state
# -------------------------------------------------------------------------

run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ],
    cwd=REPO,
)


# -------------------------------------------------------------------------
# 4. Verify remote anchor BEFORE checkout
# -------------------------------------------------------------------------

remote = subprocess.check_output(
    [
        "git",
        "rev-parse",
        "origin/main",
    ],
    cwd=REPO,
    text=True,
).strip()


print(
    "\nExpected Stage26-7C anchor:",
    EXPECTED_ANCHOR
)

print(
    "origin/main actual        :",
    remote
)


if remote != EXPECTED_ANCHOR:

    raise RuntimeError(
        "\norigin/main is NOT the expected Stage26-7C anchor.\n"
        "Do not continue automatically.\n\n"
        f"Expected: {EXPECTED_ANCHOR}\n"
        f"Actual:   {remote}"
    )


# -------------------------------------------------------------------------
# 5. Restore local main exactly
# -------------------------------------------------------------------------

run(
    [
        "git",
        "checkout",
        "-B",
        "main",
        "origin/main",
    ],
    cwd=REPO,
)


# -------------------------------------------------------------------------
# 6. Final integrity state
# -------------------------------------------------------------------------

head = subprocess.check_output(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=REPO,
    text=True,
).strip()


status = subprocess.check_output(
    [
        "git",
        "status",
        "--porcelain",
    ],
    cwd=REPO,
    text=True,
)


branch = subprocess.check_output(
    [
        "git",
        "branch",
        "--show-current",
    ],
    cwd=REPO,
    text=True,
).strip()


print(
    "\n"
    +
    "=" * 100
)

print(
    "WORKSPACE RESTORE RESULT"
)

print(
    "=" * 100
)

print(
    "Repository :",
    REPO
)

print(
    "Branch     :",
    branch
)

print(
    "HEAD       :",
    head
)

print(
    "origin/main:",
    remote
)

print(
    "Repo clean :",
    status == ""
)


if head != EXPECTED_ANCHOR:
    raise RuntimeError(
        "Local HEAD does not match the expected Stage26-7C anchor."
    )


if status:
    raise RuntimeError(
        "Repository is not clean after fresh-runtime restore."
    )


print(
    "\nRESTORE STATUS: PASS"
)

print(
    "Scientific computation performed : NO"
)

print(
    "Scientific artifact modified      : NO"
)

print(
    "Commit/push performed             : NO"
)

print(
    "GPU used                          : NO"
)

print(
    "\nNEXT:"
)

print(
    "  Workspace is ready to resume Stage26-8A."
)

STAGE26-8A-R0 :: FRESH-RUNTIME WORKSPACE RESTORE

/kaggle/working exists: True
Repository exists: False
$ git clone --branch main --single-branch https://github.com/themubasshir/ids2018-validation-safe-ablation.git /kaggle/working/ids2018-validation-safe-ablation
Cloning into '/kaggle/working/ids2018-validation-safe-ablation'...
Updating files:  23% (635/2751)
Updating files:  24% (661/2751)
Updating files:  25% (688/2751)
Updating files:  26% (716/2751)
Updating files:  27% (743/2751)
Updating files:  28% (771/2751)
Updating files:  29% (798/2751)
Updating files:  30% (826/2751)
Updating files:  31% (853/2751)
Updating files:  32% (881/2751)
Updating files:  33% (908/2751)
Updating files:  34% (936/2751)
Updating files:  35% (963/2751)
Updating files:  36% (991/2751)
Updating files:  37% (1018/2751)
Updating files:  38% (1046/2751)
Updating files:  39% (1073/2751)
Updating files:  40% (1101/2751)
Updating files:  41% (1128/2751)
Updating files:  42% (1156/2751)
Updating files:  42% (1

In [4]:
# =============================================================================
# STAGE26-8A1
# NARROW EXACT-WORKER TIMING / FINGERPRINT ORDERING DIAGNOSTIC
#
# PURPOSE
# -------
# Correct the one limitation of Stage26-8A:
#
# Stage26-8A treated worker resolution as ambiguous because its heuristic
# scoring tied worker_v1 and worker_v2.
#
# But the frozen Stage26-4C3 metadata explicitly records:
#
#   worker_sha256 =
#   ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23
#
# which uniquely identifies:
#
#   stage26_representation_worker_v2.py
#
# This cell:
#   1. proves that SHA-based worker resolution;
#   2. prints the exact relevant source with line numbers;
#   3. structurally identifies warmup/timed loops;
#   4. identifies timer start/end;
#   5. identifies output fingerprint / full-array reads;
#   6. determines whether those reads occur between timed iterations.
#
# READ ONLY.
#
# NO:
#   - representation execution
#   - timing
#   - model loading
#   - inference
#   - bootstrap
#   - Pareto
#   - PCAP
#   - Release corpus
#   - GPU
#   - Git modification
# =============================================================================

from __future__ import annotations

import ast
import hashlib
import json
import re
import subprocess
from pathlib import Path


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3"
)

EXPECTED_WORKER_SHA256 = (
    "ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23"
)

WORKER = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c1a_label_free_equivalence_erratum"
    / "stage26_representation_worker_v2.py"
)

REP_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c3_cpu1_representation_timing"
)

RECEIPT = (
    REP_DIR
    / "stage26_4c3_cpu1_representation_timing_receipt.json"
)

MANIFEST = (
    REP_DIR
    / "stage26_4c3_cpu1_representation_timing_manifest.json"
)

PLAN = (
    REP_DIR
    / "stage26_4c3_execution_plan.json"
)

RUNTIME_AUDIT = (
    Path(
        "/kaggle/working/stage26_deployment_profiling/cpu_closure"
    )
    / "stage26_8a1_exact_worker_ordering_diagnostic.json"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):
    print("\n" + "=" * 124)
    print(text)
    print("=" * 124)


def git(*args):
    p = subprocess.run(
        ["git", *args],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(p.stdout)

    return p.stdout.strip()


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(8 * 1024 * 1024)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def source_segment(lines, start, end):
    return "\n".join(
        f"{i:6d}: {lines[i - 1]}"
        for i in range(
            max(1, start),
            min(len(lines), end) + 1,
        )
    )


# =============================================================================
# 2. DURABLE STATE
# =============================================================================

banner(
    "STAGE26-8A1 :: DURABLE STATE"
)

git(
    "fetch",
    "origin",
    "main",
)

head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print("Expected parent:", EXPECTED_PARENT)
print("Local HEAD     :", head)
print("origin/main    :", remote)
print("Repo clean     :", status == "")


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Unexpected local HEAD."
    )

if remote != EXPECTED_PARENT:
    raise RuntimeError(
        "Unexpected origin/main."
    )

if status:
    raise RuntimeError(
        "Repository must be clean."
    )

if RUNTIME_AUDIT.exists():
    raise RuntimeError(
        "Transient Stage26-8A1 output already exists."
    )


# =============================================================================
# 3. SHA-BASED WORKER RESOLUTION
# =============================================================================

banner(
    "STAGE26-8A1 :: EXACT SHA-BASED WORKER RESOLUTION"
)


receipt = read_json(
    RECEIPT
)

manifest = read_json(
    MANIFEST
)

plan = read_json(
    PLAN
)


metadata_worker_shas = {
    "receipt.frozen_inputs.worker_v2_sha256":
        receipt[
            "frozen_inputs"
        ][
            "worker_v2_sha256"
        ],

    "manifest.worker_sha256":
        manifest[
            "worker_sha256"
        ],

    "plan.worker_sha256":
        plan[
            "worker_sha256"
        ],
}


for label, value in metadata_worker_shas.items():
    print(
        f"{label:44s}: {value}"
    )

    if value != EXPECTED_WORKER_SHA256:
        raise RuntimeError(
            f"{label} does not match expected worker SHA."
        )


actual_worker_sha = sha256_file(
    WORKER
)


print(
    "\nResolved worker path:"
)

print(
    " ",
    WORKER.relative_to(
        REPO
    )
)

print(
    "Resolved worker SHA256:"
)

print(
    " ",
    actual_worker_sha
)


if actual_worker_sha != EXPECTED_WORKER_SHA256:
    raise RuntimeError(
        "Resolved worker file does not match frozen 4C3 worker SHA."
    )


# Check uniqueness across every tracked Python file.
matching_python_files = []


for relpath in git(
    "ls-files",
    "*.py",
).splitlines():

    path = REPO / relpath

    if (
        path.is_file()
        and
        sha256_file(path)
        ==
        EXPECTED_WORKER_SHA256
    ):
        matching_python_files.append(
            relpath
        )


print(
    "\nTracked Python files with exact frozen worker SHA:"
)

for item in matching_python_files:
    print(
        " ",
        item
    )


if matching_python_files != [
    str(
        WORKER.relative_to(
            REPO
        )
    )
]:
    raise RuntimeError(
        "Frozen worker SHA does not resolve uniquely."
    )


print(
    "\nWORKER RESOLUTION: UNIQUE_AND_CERTIFIED"
)


# =============================================================================
# 4. LOAD / PARSE EXACT SOURCE
# =============================================================================

banner(
    "STAGE26-8A1 :: EXACT WORKER SOURCE STRUCTURE"
)


source_text = WORKER.read_text(
    encoding="utf-8"
)

lines = source_text.splitlines()

tree = ast.parse(
    source_text
)


print(
    "Source lines:",
    len(lines)
)


# =============================================================================
# 5. FIND FUNCTIONS
# =============================================================================

functions = []


for node in ast.walk(tree):

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    ):

        functions.append(
            {
                "name":
                    node.name,

                "lineno":
                    node.lineno,

                "end_lineno":
                    node.end_lineno,
            }
        )


functions.sort(
    key=lambda x: x[
        "lineno"
    ]
)


print(
    "\nFunctions:"
)

for fn in functions:
    print(
        f"  {fn['lineno']:5d}-{fn['end_lineno']:5d} "
        f"{fn['name']}"
    )


# =============================================================================
# 6. FIND ALL SEMANTICALLY RELEVANT LINES
# =============================================================================

banner(
    "STAGE26-8A1 :: TIMER / LOOP / FINGERPRINT LINES"
)


patterns = {
    "timer":
        re.compile(
            r"perf_counter_ns|perf_counter",
            re.I,
        ),

    "fingerprint":
        re.compile(
            r"fingerprint",
            re.I,
        ),

    "hash":
        re.compile(
            r"sha256|hashlib|digest|hexdigest|tobytes",
            re.I,
        ),

    "timed_runs":
        re.compile(
            r"timed_runs",
            re.I,
        ),

    "warmup_runs":
        re.compile(
            r"warmup_runs",
            re.I,
        ),

    "materialize":
        re.compile(
            r"materialize",
            re.I,
        ),

    "for_loop":
        re.compile(
            r"^\s*for\s+",
        ),
}


hits = {
    key: []
    for key in patterns
}


for number, line in enumerate(
    lines,
    start=1,
):

    for key, pattern in patterns.items():

        if pattern.search(line):
            hits[
                key
            ].append(
                number
            )


for key, values in hits.items():
    print(
        f"{key:16s}:",
        values
    )


# =============================================================================
# 7. AST — LOCATE LOOPS CONTAINING TIMERS
# =============================================================================

banner(
    "STAGE26-8A1 :: REPEATED TIMING LOOP STRUCTURE"
)


timing_loops = []


for node in ast.walk(tree):

    if not isinstance(
        node,
        (
            ast.For,
            ast.While,
        ),
    ):
        continue


    start = node.lineno
    end = node.end_lineno


    timer_lines = [
        n
        for n in hits[
            "timer"
        ]
        if start <= n <= end
    ]

    fp_lines = sorted(
        set(
            [
                n
                for n in hits[
                    "fingerprint"
                ]
                if start <= n <= end
            ]
            +
            [
                n
                for n in hits[
                    "hash"
                ]
                if start <= n <= end
            ]
        )
    )


    if timer_lines:

        try:
            loop_header = ast.unparse(
                node
            ).splitlines()[0]

        except Exception:
            loop_header = lines[
                start - 1
            ].strip()


        timing_loops.append(
            {
                "start":
                    start,

                "end":
                    end,

                "header":
                    loop_header,

                "timer_lines":
                    timer_lines,

                "fingerprint_hash_lines":
                    fp_lines,
            }
        )


for index, item in enumerate(
    timing_loops,
    start=1,
):

    print(
        f"\nTIMING LOOP {index}"
    )

    print(
        "  lines       :",
        item[
            "start"
        ],
        "-",
        item[
            "end"
        ]
    )

    print(
        "  timer lines :",
        item[
            "timer_lines"
        ]
    )

    print(
        "  hash/fp     :",
        item[
            "fingerprint_hash_lines"
        ]
    )

    print(
        "\n"
        +
        source_segment(
            lines,
            item[
                "start"
            ],
            item[
                "end"
            ],
        )
    )


# =============================================================================
# 8. AST CALL ORDER WITHIN TIMING LOOPS
# =============================================================================

banner(
    "STAGE26-8A1 :: CALL ORDER WITHIN TIMING LOOPS"
)


call_records = []


for loop_index, loop_info in enumerate(
    timing_loops,
    start=1,
):

    start = loop_info[
        "start"
    ]

    end = loop_info[
        "end"
    ]


    loop_node = None


    for node in ast.walk(tree):

        if (
            isinstance(
                node,
                (
                    ast.For,
                    ast.While,
                ),
            )
            and
            node.lineno == start
            and
            node.end_lineno == end
        ):
            loop_node = node
            break


    if loop_node is None:
        raise RuntimeError(
            "Could not recover timing-loop AST node."
        )


    local_calls = []


    for subnode in ast.walk(
        loop_node
    ):

        if not isinstance(
            subnode,
            ast.Call,
        ):
            continue


        try:
            call_name = ast.unparse(
                subnode.func
            )

        except Exception:
            call_name = "<unparse-failed>"


        local_calls.append(
            {
                "line":
                    subnode.lineno,

                "call":
                    call_name,
            }
        )


    local_calls.sort(
        key=lambda x: (
            x[
                "line"
            ],
            x[
                "call"
            ],
        )
    )


    print(
        f"\nTIMING LOOP {loop_index} CALLS:"
    )


    for record in local_calls:
        print(
            f"  {record['line']:5d}: "
            f"{record['call']}"
        )


    call_records.append(
        {
            "loop_index":
                loop_index,

            "loop_start":
                start,

            "loop_end":
                end,

            "calls":
                local_calls,
        }
    )


# =============================================================================
# 9. EXACT TIMER / FINGERPRINT ORDER CLASSIFICATION
# =============================================================================

banner(
    "STAGE26-8A1 :: EXACT ORDER CLASSIFICATION"
)


classification_evidence = []

classification = (
    "NOT_RESOLVABLE_FROM_EXACT_WORKER"
)


for loop in timing_loops:

    timer_lines = sorted(
        loop[
            "timer_lines"
        ]
    )


    if len(timer_lines) < 2:
        continue


    timer_start = timer_lines[
        0
    ]

    timer_stop = timer_lines[
        1
    ]


    fp_hash_lines = sorted(
        loop[
            "fingerprint_hash_lines"
        ]
    )


    inside_timer = [
        line
        for line in fp_hash_lines
        if timer_start
        <
        line
        <
        timer_stop
    ]


    after_timer_before_loop_end = [
        line
        for line in fp_hash_lines
        if timer_stop
        <
        line
        <=
        loop[
            "end"
        ]
    ]


    evidence = {
        "loop_start":
            loop[
                "start"
            ],

        "loop_end":
            loop[
                "end"
            ],

        "timer_start_line":
            timer_start,

        "timer_stop_line":
            timer_stop,

        "fingerprint_hash_inside_timer":
            inside_timer,

        "fingerprint_hash_after_timer_before_next_iteration":
            after_timer_before_loop_end,
    }


    classification_evidence.append(
        evidence
    )


    print(
        json.dumps(
            evidence,
            indent=2,
            sort_keys=True,
        )
    )


    if after_timer_before_loop_end:

        classification = (
            "OUTSIDE_TIMER_BEFORE_NEXT_TIMED_ITERATION"
        )

        break


    if inside_timer:

        classification = (
            "INSIDE_TIMER"
        )

        break


# If there is no fingerprint/hash inside a repeated timed loop,
# inspect whether hashing occurs only after the timing loops finish.

if classification == (
    "NOT_RESOLVABLE_FROM_EXACT_WORKER"
):

    if timing_loops:

        last_timing_loop_end = max(
            item[
                "end"
            ]
            for item in timing_loops
        )


        all_fp_hash = sorted(
            set(
                hits[
                    "fingerprint"
                ]
                +
                hits[
                    "hash"
                ]
            )
        )


        relevant_after = [
            n
            for n in all_fp_hash
            if n > last_timing_loop_end
        ]


        relevant_before_or_inside = [
            n
            for n in all_fp_hash
            if n <= last_timing_loop_end
        ]


        if (
            relevant_after
            and
            not relevant_before_or_inside
        ):

            classification = (
                "OUTSIDE_TIMER_AFTER_ALL_TIMED_ITERATIONS"
            )


print(
    "\nFINAL EXACT-WORKER CLASSIFICATION:"
)

print(
    " ",
    classification
)


# =============================================================================
# 10. PRINT ALL CRITICAL SOURCE CONTEXT
# =============================================================================

banner(
    "STAGE26-8A1 :: CRITICAL SOURCE CONTEXT"
)


critical_lines = sorted(
    set(
        hits[
            "timer"
        ]
        +
        hits[
            "fingerprint"
        ]
        +
        hits[
            "hash"
        ]
        +
        hits[
            "timed_runs"
        ]
        +
        hits[
            "warmup_runs"
        ]
    )
)


printed_ranges = []


for line_number in critical_lines:

    start = max(
        1,
        line_number - 6,
    )

    end = min(
        len(lines),
        line_number + 8,
    )


    if any(
        start >= old_start
        and
        end <= old_end
        for old_start, old_end in printed_ranges
    ):
        continue


    printed_ranges.append(
        (
            start,
            end,
        )
    )


    print(
        "\n"
        +
        "-" * 124
    )

    print(
        f"Worker lines {start}-{end}"
    )

    print(
        "-" * 124
    )

    print(
        source_segment(
            lines,
            start,
            end,
        )
    )


# =============================================================================
# 11. SCIENTIFIC CONSEQUENCE
# =============================================================================

banner(
    "STAGE26-8A1 :: SCIENTIFIC CONSEQUENCE"
)


if classification == (
    "OUTSIDE_TIMER_BEFORE_NEXT_TIMED_ITERATION"
):

    consequence = (
        "CONFIRMED: the exact frozen Stage26-4C3 worker performs an "
        "integrity/fingerprint-related full-output read after the timer stops "
        "but before control returns to the next iteration of the repeated "
        "timing loop. The read is therefore excluded from the reported "
        "latency itself but may perturb cache/CPU state entering the next "
        "timed observation."
    )

    next_action = (
        "FREEZE_PROSPECTIVE_STAGE26_8B_4C3_SENSITIVITY_PROTOCOL"
    )


elif classification == "INSIDE_TIMER":

    consequence = (
        "CONFIRMED: integrity/fingerprint work occurs inside the measured "
        "timer boundary. A prospective corrected/sensitivity implementation "
        "must be frozen before any new measurement."
    )

    next_action = (
        "FREEZE_PROSPECTIVE_STAGE26_8B_4C3_SENSITIVITY_PROTOCOL"
    )


elif classification == (
    "OUTSIDE_TIMER_AFTER_ALL_TIMED_ITERATIONS"
):

    consequence = (
        "The exact frozen worker performs integrity/fingerprint work only "
        "after all repeated timed observations. The hypothesized "
        "inter-iteration cache-state contamination is therefore not "
        "supported."
    )

    next_action = (
        "ANCHOR_NO_4C3_SENSITIVITY_RUN_REQUIRED"
    )


else:

    consequence = (
        "Even after exact SHA-based worker resolution, ordering cannot yet "
        "be certified automatically. A line-specific diagnostic would be "
        "required before any sensitivity measurement decision."
    )

    next_action = (
        "LINE_SPECIFIC_DIAGNOSTIC_REQUIRED"
    )


print(
    consequence
)

print(
    "\nNext action:"
)

print(
    " ",
    next_action
)


# =============================================================================
# 12. WRITE TRANSIENT DIAGNOSTIC
# =============================================================================

RUNTIME_AUDIT.parent.mkdir(
    parents=True,
    exist_ok=True,
)


payload = {
    "schema":
        "stage26_8a1_exact_worker_ordering_diagnostic_v1",

    "scientific_parent":
        EXPECTED_PARENT,

    "status":
        "PASS_EXACT_SHA_BASED_WORKER_DIAGNOSTIC",

    "worker_resolution": {
        "classification":
            "UNIQUE_AND_CERTIFIED",

        "repo_relative_path":
            str(
                WORKER.relative_to(
                    REPO
                )
            ),

        "sha256":
            actual_worker_sha,

        "metadata_worker_shas":
            metadata_worker_shas,

        "tracked_python_files_with_matching_sha":
            matching_python_files,
    },

    "timing_loops":
        timing_loops,

    "call_records":
        call_records,

    "classification_evidence":
        classification_evidence,

    "classification":
        classification,

    "scientific_consequence":
        consequence,

    "next_action":
        next_action,

    "scientific_state": {
        "worker_uniquely_resolved":
            True,

        "representation_executed":
            False,

        "timing_executed":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "bootstrap_executed":
            False,

        "Pareto_recomputed":
            False,

        "PCAP_accessed":
            False,

        "Release_corpus_accessed":
            False,

        "GPU_used":
            False,

        "Git_modified":
            False,
    },
}


with RUNTIME_AUDIT.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        payload,
        f,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    )

    f.write(
        "\n"
    )


audit_sha = sha256_file(
    RUNTIME_AUDIT
)


print(
    "\nTransient output:"
)

print(
    " ",
    RUNTIME_AUDIT
)

print(
    "SHA256:"
)

print(
    " ",
    audit_sha
)


# =============================================================================
# 13. FINAL READ-ONLY GIT GATE
# =============================================================================

banner(
    "STAGE26-8A1 COMPLETE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:
    raise RuntimeError(
        "HEAD changed during diagnostic."
    )


if final_remote != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main changed during diagnostic."
    )


if final_status:
    raise RuntimeError(
        "Diagnostic unexpectedly modified Git."
    )


print(
    "\nWORKER:"
)

print(
    " ",
    WORKER.relative_to(
        REPO
    )
)

print(
    " ",
    actual_worker_sha
)


print(
    "\nCLASSIFICATION:"
)

print(
    " ",
    classification
)


print(
    "\nNEXT:"
)

print(
    " ",
    next_action
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  worker ambiguity                : RESOLVED"
)

print(
    "  original 4C3 modified          : NO"
)

print(
    "  representation executed        : NO"
)

print(
    "  sensitivity timing             : NO"
)

print(
    "  Stage26-7 ratios recomputed    : NO"
)

print(
    "  Pareto recomputed              : NO"
)

print(
    "  GPU                            : NO"
)

print(
    "  Git modified                   : NO"
)


STAGE26-8A1 :: DURABLE STATE
Expected parent: 9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3
Local HEAD     : 9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3
origin/main    : 9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3
Repo clean     : True

STAGE26-8A1 :: EXACT SHA-BASED WORKER RESOLUTION
receipt.frozen_inputs.worker_v2_sha256      : ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23
manifest.worker_sha256                      : ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23
plan.worker_sha256                          : ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23

Resolved worker path:
  results/stage26_deployment_profiling/stage26_4c1a_label_free_equivalence_erratum/stage26_representation_worker_v2.py
Resolved worker SHA256:
  ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23

Tracked Python files with exact frozen worker SHA:
  results/stage26_deployment_profiling/stage26_4c1a_label_free_equivalence_erratum/stage26_representation_

In [5]:
# =============================================================================
# STAGE26-8B
# FREEZE PROSPECTIVE PAIRED STAGE26-4C3 INTEGRITY-READ SENSITIVITY PROTOCOL
# + CLEAN V3 WORKER
# COMMIT + PUSH + REMOTE BYTE VERIFY
#
# DURABLE SCIENTIFIC PARENT:
#   9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3
#
# CONFIRMED STAGE26-8A1 FINDING
# --------------------------------
# Exact historical Stage26-4C3 worker:
#
#   stage26_representation_worker_v2.py
#
# SHA256:
#   ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23
#
# Exact ordering:
#
#   timer start
#   materialize_batch(...)
#   timer stop
#   fingerprint(images, masks)
#   ...
#   next timed iteration
#
# Classification:
#
#   OUTSIDE_TIMER_BEFORE_NEXT_TIMED_ITERATION
#
#
# WHY A PAIRED SENSITIVITY DESIGN
# --------------------------------
# We do NOT compare a new V3 timing directly against historical 4C3 and call
# the difference a fingerprint effect, because Kaggle runtime/hardware state
# may differ from the historical session.
#
# Instead, on ONE future runtime we will run:
#
#   A = exact frozen historical worker V2
#   B = clean sensitivity worker V3
#
# under the same runtime, inputs, CPU1 affinity, batch, warmups, timed runs,
# and fresh-process execution.
#
# Thus the sensitivity estimand is V3 versus V2 measured contemporaneously.
#
#
# V3 CHANGE — AND ONLY INTENDED CHANGE
# -------------------------------------
# V2:
#
#   each timed iteration:
#       time materialize_batch
#       fingerprint full image + mask
#       next timed iteration
#
# V3:
#
#   each timed iteration:
#       time materialize_batch
#       NO full-output fingerprint read
#       next timed iteration
#
#   after ALL timed iterations:
#       fingerprint final image + mask once
#       compare with final warmup fingerprint
#
# Warmup behavior remains unchanged.
# materialize_batch implementation remains unchanged.
# timer boundaries remain unchanged.
# output shapes/types remain unchanged.
#
#
# SENSITIVITY DESIGN
# ------------------
# Hardware:
#   CPU1 only.
#
# Frozen batches:
#   [1, 64, 256, 1024, 8192]
#
# Frozen warmup/timed-run counts:
#   inherited EXACTLY from committed historical Stage26-4C3 configs.
#
# Pair replicates:
#   5 per batch.
#
# Total pair blocks:
#   25.
#
# Total fresh worker processes:
#   50.
#
# Randomization:
#   np.random.default_rng(26042), PCG64.
#
#   - shuffle the 25 pair blocks;
#   - independently randomize V2/V3 order inside every pair.
#
# No replacement pair is invented after seeing results.
#
#
# PRIMARY SENSITIVITY ESTIMAND
# ----------------------------
# For each pair:
#
#   throughput_ratio_clean_over_original
#       = median_throughput_V3 / median_throughput_V2
#
# Secondary descriptive estimands:
#
#   p50_latency_ratio_clean_over_original
#       = p50_V3 / p50_V2
#
#   p95_latency_ratio_clean_over_original
#       = p95_V3 / p95_V2
#
#   p99 ratio when original n >= 100.
#
# Across the five pairs at each batch:
#
#   report all five pair ratios
#   report median pair ratio
#   report min/max pair ratio
#
# NO hypothesis test.
# NO significance threshold.
# NO post-hoc "materiality" cutoff.
# NO bootstrap.
#
#
# SCIENTIFIC POLICY
# -----------------
# This sensitivity experiment DOES NOT replace Stage26-4C3.
#
# Stage26-4C3 remains the original historical measurement.
# Stage26-7C remains unchanged.
# Stage26-7C component-capacity ratios are NOT recomputed here.
#
# The sensitivity result is robustness evidence for publication framing.
#
# If future publication text discusses the 4C3 component result, the original
# measurement and paired sensitivity evidence must be distinguished.
#
#
# THIS CELL DOES NOT EXECUTE THE SENSITIVITY EXPERIMENT.
#
# NO:
#   - representation timing
#   - worker execution
#   - corpus access
#   - inference
#   - model loading
#   - bootstrap
#   - Pareto recomputation
#   - Stage26-7 recomputation
#   - PCAP
#   - GPU
# =============================================================================

from __future__ import annotations

import ast
import difflib
import hashlib
import json
import os
import stat
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone

import numpy as np


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3"
)

COMMIT_SUBJECT = (
    "stage26: freeze paired representation sensitivity protocol"
)


# -----------------------------------------------------------------------------
# Stage26-8A / 8A1 transient provenance
# -----------------------------------------------------------------------------

RUNTIME_CLOSURE = Path(
    "/kaggle/working/stage26_deployment_profiling/cpu_closure"
)

AUDIT_8A = (
    RUNTIME_CLOSURE
    / "stage26_8a_4c3_integrity_read_implementation_audit.json"
)

AUDIT_8A1 = (
    RUNTIME_CLOSURE
    / "stage26_8a1_exact_worker_ordering_diagnostic.json"
)

EXPECTED_8A_SHA256 = (
    "0d457c476567426c0093d29222450f411e94732455139a5d2f39ad2c0d1ed73c"
)

EXPECTED_8A1_SHA256 = (
    "e2121362d23a8e81afd367d42df944f8558422c0984762c5fbca5dd5f186a9c0"
)


# -----------------------------------------------------------------------------
# Exact historical V2 worker
# -----------------------------------------------------------------------------

V2_WORKER = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c1a_label_free_equivalence_erratum"
    / "stage26_representation_worker_v2.py"
)

EXPECTED_V2_SHA256 = (
    "ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23"
)


# -----------------------------------------------------------------------------
# Historical Stage26-4C3
# -----------------------------------------------------------------------------

C4_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c3_cpu1_representation_timing"
)

C4_SUMMARY_CSV = (
    C4_DIR
    / "stage26_4c3_cpu1_representation_summary.csv"
)

C4_SUMMARY_JSON = (
    C4_DIR
    / "stage26_4c3_cpu1_representation_summary.json"
)

C4_RECEIPT = (
    C4_DIR
    / "stage26_4c3_cpu1_representation_timing_receipt.json"
)

C4_MANIFEST = (
    C4_DIR
    / "stage26_4c3_cpu1_representation_timing_manifest.json"
)

C4_PLAN = (
    C4_DIR
    / "stage26_4c3_execution_plan.json"
)

C4_RAW = (
    C4_DIR
    / "stage26_4c3_raw_observations.csv"
)


EXPECTED_C4_SUMMARY_CSV_SHA256 = (
    "ca5bbc17edfa4bb088599236b4099c1986fd73cb7f087ef3ac753a524afd2ba5"
)

EXPECTED_C4_SUMMARY_JSON_SHA256 = (
    "63c8041144668c62218fc9c46eb6a57aba9dcda0c9a3f0477f932f6a572c34f7"
)

EXPECTED_C4_RECEIPT_SHA256 = (
    "ed55b65905080dd6b796a54fb720de76bf100fdc7f3271a973e49c81f1722b14"
)

EXPECTED_C4_MANIFEST_SHA256 = (
    "bfbc9000a708a803c53a11b591e8021d54325d1b7bccebb4b0aad65e1439b9db"
)

EXPECTED_C4_PLAN_SHA256 = (
    "5e23b40c0e5d13a91641c42ad3ac95b4f3a828a6e351a3cd139d826d3ade2568"
)

EXPECTED_C4_RAW_SHA256 = (
    "189852745ef83330526f6c680d459c7075a695f97614aa6c76a7ae7ded3447b3"
)


# -----------------------------------------------------------------------------
# Stage26-7C — immutable
# -----------------------------------------------------------------------------

C7_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_7c_cpu_capacity_scaling"
)

C7_RESULTS = (
    C7_DIR
    / "stage26_7c_capacity_scaling_results.json"
)

C7_RECEIPT = (
    C7_DIR
    / "stage26_7c_capacity_scaling_receipt.json"
)

EXPECTED_C7_RESULTS_SHA256 = (
    "ee5a852fb43ce01c980e64ef83611a88808d3a11a56a078609467fb899e39fdf"
)

EXPECTED_C7_RECEIPT_SHA256 = (
    "30bfd4657a5a0218ce22a4fab1922ee26e07496c148809ec04c8059656b73ce4"
)


# -----------------------------------------------------------------------------
# New Stage26-8B package
# -----------------------------------------------------------------------------

CHECKPOINT_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_8b_representation_sensitivity_protocol_lock"
)

CHECKPOINT_DIR = (
    REPO
    / CHECKPOINT_REL
)

PROVENANCE_DIR = (
    CHECKPOINT_DIR
    / "provenance"
)

PROTOCOL = (
    CHECKPOINT_DIR
    / "stage26_8b_representation_sensitivity_protocol.json"
)

EXECUTION_PLAN = (
    CHECKPOINT_DIR
    / "stage26_8b_representation_sensitivity_execution_plan.json"
)

V3_WORKER = (
    CHECKPOINT_DIR
    / "stage26_representation_worker_v3_sensitivity.py"
)

V2_TO_V3_DIFF = (
    CHECKPOINT_DIR
    / "stage26_8b_v2_to_v3_worker.diff"
)

FREEZE_RECEIPT = (
    CHECKPOINT_DIR
    / "stage26_8b_representation_sensitivity_freeze_receipt.json"
)

MANIFEST = (
    CHECKPOINT_DIR
    / "stage26_8b_representation_sensitivity_lock_manifest.json"
)

DURABLE_8A = (
    PROVENANCE_DIR
    / "stage26_8a_4c3_integrity_read_implementation_audit.json"
)

DURABLE_8A1 = (
    PROVENANCE_DIR
    / "stage26_8a1_exact_worker_ordering_diagnostic.json"
)


BATCHES = [
    1,
    64,
    256,
    1024,
    8192,
]

PAIR_REPLICATES = 5
RANDOMIZATION_SEED = 26042


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
        text=True,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_json(
    path,
    payload,
):

    path = Path(path)

    tmp = Path(
        str(path)
        +
        ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def atomic_text(
    path,
    text,
):

    path = Path(path)

    tmp = Path(
        str(path)
        +
        ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as f:

        f.write(text)

        if not text.endswith(
            "\n"
        ):
            f.write(
                "\n"
            )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        text=False,
    )

    return bytes(
        p.stdout
    )


# =============================================================================
# 2. DURABLE GIT GATE
# =============================================================================

banner(
    "STAGE26-8B :: DURABLE SCIENTIFIC PARENT"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-8B parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Stage26-8B."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if CHECKPOINT_DIR.exists():

    raise RuntimeError(
        "Stage26-8B checkpoint already exists."
    )


# =============================================================================
# 3. EXACT SOURCE / PROVENANCE IDENTITY
# =============================================================================

banner(
    "STAGE26-8B :: EXACT SOURCE / PROVENANCE IDENTITY"
)


checks = [
    (
        "8A transient audit",
        AUDIT_8A,
        EXPECTED_8A_SHA256,
    ),

    (
        "8A1 exact-worker audit",
        AUDIT_8A1,
        EXPECTED_8A1_SHA256,
    ),

    (
        "historical worker V2",
        V2_WORKER,
        EXPECTED_V2_SHA256,
    ),

    (
        "4C3 summary CSV",
        C4_SUMMARY_CSV,
        EXPECTED_C4_SUMMARY_CSV_SHA256,
    ),

    (
        "4C3 summary JSON",
        C4_SUMMARY_JSON,
        EXPECTED_C4_SUMMARY_JSON_SHA256,
    ),

    (
        "4C3 receipt",
        C4_RECEIPT,
        EXPECTED_C4_RECEIPT_SHA256,
    ),

    (
        "4C3 manifest",
        C4_MANIFEST,
        EXPECTED_C4_MANIFEST_SHA256,
    ),

    (
        "4C3 execution plan",
        C4_PLAN,
        EXPECTED_C4_PLAN_SHA256,
    ),

    (
        "4C3 raw observations",
        C4_RAW,
        EXPECTED_C4_RAW_SHA256,
    ),

    (
        "7C results",
        C7_RESULTS,
        EXPECTED_C7_RESULTS_SHA256,
    ),

    (
        "7C receipt",
        C7_RECEIPT,
        EXPECTED_C7_RECEIPT_SHA256,
    ),
]


identity = {}


for label, path, expected in checks:

    if not Path(path).is_file():

        raise FileNotFoundError(
            path
        )


    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:28s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Identity mismatch: {label}"
        )


    identity[
        label
    ] = {
        "path":
            str(path),

        "sha256":
            actual,
    }


# =============================================================================
# 4. EXACT 8A1 CLASSIFICATION GATE
# =============================================================================

banner(
    "STAGE26-8B :: EXACT 8A1 FINDING GATE"
)


audit_8a1 = json.loads(
    AUDIT_8A1.read_text(
        encoding="utf-8"
    )
)


if audit_8a1[
    "worker_resolution"
][
    "classification"
] != "UNIQUE_AND_CERTIFIED":

    raise RuntimeError(
        "8A1 worker resolution is not certified."
    )


if audit_8a1[
    "worker_resolution"
][
    "sha256"
] != EXPECTED_V2_SHA256:

    raise RuntimeError(
        "8A1 worker SHA mismatch."
    )


if audit_8a1[
    "classification"
] != "OUTSIDE_TIMER_BEFORE_NEXT_TIMED_ITERATION":

    raise RuntimeError(
        "Unexpected 8A1 ordering classification."
    )


if audit_8a1[
    "next_action"
] != "FREEZE_PROSPECTIVE_STAGE26_8B_4C3_SENSITIVITY_PROTOCOL":

    raise RuntimeError(
        "Unexpected 8A1 next action."
    )


print(
    "Worker                 : UNIQUE_AND_CERTIFIED"
)

print(
    "Worker SHA             :",
    EXPECTED_V2_SHA256
)

print(
    "Ordering classification:",
    audit_8a1[
        "classification"
    ]
)


# =============================================================================
# 5. VERIFY HISTORICAL METADATA ALSO IDENTIFIES V2
# =============================================================================

banner(
    "STAGE26-8B :: HISTORICAL WORKER PROVENANCE"
)


c4_receipt = json.loads(
    C4_RECEIPT.read_text(
        encoding="utf-8"
    )
)

c4_manifest = json.loads(
    C4_MANIFEST.read_text(
        encoding="utf-8"
    )
)

c4_plan = json.loads(
    C4_PLAN.read_text(
        encoding="utf-8"
    )
)


historical_worker_shas = {
    "receipt.worker_v2_sha256":
        c4_receipt[
            "frozen_inputs"
        ][
            "worker_v2_sha256"
        ],

    "manifest.worker_sha256":
        c4_manifest[
            "worker_sha256"
        ],

    "execution_plan.worker_sha256":
        c4_plan[
            "worker_sha256"
        ],
}


for name, value in historical_worker_shas.items():

    print(
        f"{name:38s}: "
        f"{value}"
    )


    if value != EXPECTED_V2_SHA256:

        raise RuntimeError(
            f"Historical worker provenance mismatch: {name}"
        )


# =============================================================================
# 6. RESOLVE EXACT HISTORICAL CONDITION CONFIGS
# =============================================================================

banner(
    "STAGE26-8B :: HISTORICAL CONDITION CONFIG FREEZE"
)


condition_configs = {}


for batch in BATCHES:

    config_candidates = sorted(
        C4_DIR.glob(
            f"conditions/batch_{batch}/*_config.json"
        )
    )


    if len(
        config_candidates
    ) != 1:

        raise RuntimeError(
            f"B={batch}: expected one committed historical config, "
            f"found {len(config_candidates)}."
        )


    config_path = config_candidates[
        0
    ]

    config = json.loads(
        config_path.read_text(
            encoding="utf-8"
        )
    )


    if int(
        config[
            "batch_size"
        ]
    ) != batch:

        raise RuntimeError(
            f"B={batch}: config batch_size mismatch."
        )


    condition_configs[
        str(batch)
    ] = {
        "repo_relative_path":
            str(
                config_path.relative_to(
                    REPO
                )
            ),

        "sha256":
            sha256_file(
                config_path
            ),

        "config":
            config,
    }


    print(
        f"\nB={batch}"
    )

    print(
        "  path  :",
        config_path.relative_to(
            REPO
        )
    )

    print(
        "  SHA256:",
        condition_configs[
            str(batch)
        ][
            "sha256"
        ]
    )

    print(
        "  config:"
    )

    print(
        json.dumps(
            config,
            indent=2,
            sort_keys=True,
        )
    )


# =============================================================================
# 7. CREATE CHECKPOINT DIRECTORIES
# =============================================================================

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

PROVENANCE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# =============================================================================
# 8. DURABILIZE 8A / 8A1 PROVENANCE
# =============================================================================

banner(
    "STAGE26-8B :: DURABILIZE AUDIT PROVENANCE"
)


DURABLE_8A.write_bytes(
    AUDIT_8A.read_bytes()
)

DURABLE_8A1.write_bytes(
    AUDIT_8A1.read_bytes()
)


if sha256_file(
    DURABLE_8A
) != EXPECTED_8A_SHA256:

    raise RuntimeError(
        "Durable Stage26-8A copy mismatch."
    )


if sha256_file(
    DURABLE_8A1
) != EXPECTED_8A1_SHA256:

    raise RuntimeError(
        "Durable Stage26-8A1 copy mismatch."
    )


print(
    "8A  :",
    EXPECTED_8A_SHA256
)

print(
    "8A1 :",
    EXPECTED_8A1_SHA256
)


# =============================================================================
# 9. BUILD CLEAN V3 FROM EXACT V2 — SOURCE TRANSFORMATION ONLY
# =============================================================================

banner(
    "STAGE26-8B :: BUILD CLEAN V3 FROM EXACT FROZEN V2"
)


v2_text = V2_WORKER.read_text(
    encoding="utf-8"
)

v2_lines = v2_text.splitlines()


# Exact historical source was already audited:
# timed loop = lines 682..775
# timer       = 687..695
# fingerprint = 710..724
#
# Remove the per-timed-iteration fingerprint block, then insert one
# integrity fingerprint after the entire timed loop.


REMOVE_START = 709
REMOVE_END = 729
TIMED_LOOP_END = 775


expected_remove_fragment = "\n".join(
    v2_lines[
        REMOVE_START - 1:
        REMOVE_END
    ]
)


if (
    "Integrity check intentionally outside timer."
    not in
    expected_remove_fragment
):

    raise RuntimeError(
        "Expected V2 integrity block not found at audited line range."
    )


if "fingerprint(" not in expected_remove_fragment:

    raise RuntimeError(
        "Expected V2 fingerprint call missing from audited block."
    )


# Delete historical per-iteration integrity block.
v3_lines = (
    v2_lines[
        :REMOVE_START - 1
    ]
    +
    v2_lines[
        REMOVE_END:
    ]
)


# After deletion, the original timed loop end shifts upward.
removed_count = (
    REMOVE_END
    -
    REMOVE_START
    +
    1
)

new_loop_end_index_1based = (
    TIMED_LOOP_END
    -
    removed_count
)


post_loop_block = [
    "",
    "    # Stage26-8B sensitivity V3:",
    "    # perform full-output integrity read only AFTER all timed iterations.",
    "    # This removes the historical inter-iteration fingerprint/cache-state",
    "    # perturbation while preserving the exact materialization timer boundary.",
    "    if not observations:",
    "",
    "        raise RuntimeError(",
    '            "no timed observations produced"',
    "        )",
    "",
    "    reference_fingerprint = fingerprint(",
    "        images,",
    "        masks,",
    "    )",
    "",
    "    if (",
    "        warm_fingerprint is not None",
    "        and",
    "        reference_fingerprint",
    "        !=",
    "        warm_fingerprint",
    "    ):",
    "",
    "        raise RuntimeError(",
    '            "final timed representation differs from final warmup representation"',
    "        )",
    "",
]


v3_lines = (
    v3_lines[
        :new_loop_end_index_1based
    ]
    +
    post_loop_block
    +
    v3_lines[
        new_loop_end_index_1based:
    ]
)


v3_text = "\n".join(
    v3_lines
) + "\n"


# Syntax must remain valid before writing.
ast.parse(
    v3_text
)


atomic_text(
    V3_WORKER,
    v3_text,
)


v3_sha = sha256_file(
    V3_WORKER
)


print(
    "V2 SHA256:",
    EXPECTED_V2_SHA256
)

print(
    "V3 SHA256:",
    v3_sha
)


if v3_sha == EXPECTED_V2_SHA256:

    raise RuntimeError(
        "Sensitivity V3 unexpectedly identical to V2."
    )


# =============================================================================
# 10. WRITE EXACT V2 -> V3 DIFF
# =============================================================================

v3_diff = "".join(
    difflib.unified_diff(
        v2_text.splitlines(
            keepends=True
        ),
        v3_text.splitlines(
            keepends=True
        ),
        fromfile=(
            "stage26_representation_worker_v2.py"
        ),
        tofile=(
            "stage26_representation_worker_v3_sensitivity.py"
        ),
    )
)


atomic_text(
    V2_TO_V3_DIFF,
    v3_diff,
)


diff_sha = sha256_file(
    V2_TO_V3_DIFF
)


print(
    "\nV2 -> V3 diff SHA256:",
    diff_sha
)

print(
    "\nExact diff:"
)

print(
    v3_diff
)


# =============================================================================
# 11. STATIC V3 STRUCTURAL SAFETY AUDIT
# =============================================================================

banner(
    "STAGE26-8B :: STATIC V3 TIMING-LOOP AUDIT"
)


v3_tree = ast.parse(
    v3_text
)

v3_source_lines = v3_text.splitlines()


timing_loops = []


for node in ast.walk(
    v3_tree
):

    if not isinstance(
        node,
        (
            ast.For,
            ast.While,
        ),
    ):

        continue


    segment = "\n".join(
        v3_source_lines[
            node.lineno - 1:
            node.end_lineno
        ]
    )


    if (
        "perf_counter_ns"
        in
        segment
    ):

        timing_loops.append(
            node
        )


if len(
    timing_loops
) != 1:

    raise RuntimeError(
        f"Expected exactly one V3 timed loop; found {len(timing_loops)}."
    )


timing_loop = timing_loops[
    0
]

timing_segment = "\n".join(
    v3_source_lines[
        timing_loop.lineno - 1:
        timing_loop.end_lineno
    ]
)


print(
    "V3 timing loop lines:",
    timing_loop.lineno,
    "-",
    timing_loop.end_lineno
)


print(
    "\n"
    +
    timing_segment
)


if timing_segment.count(
    "perf_counter_ns"
) != 2:

    raise RuntimeError(
        "V3 timer boundary changed unexpectedly."
    )


if "source.materialize_batch(" not in timing_segment:

    raise RuntimeError(
        "V3 timed loop no longer materializes representation."
    )


if "fingerprint(" in timing_segment:

    raise RuntimeError(
        "V3 still fingerprints inside repeated timed loop."
    )


if "hashlib" in timing_segment:

    raise RuntimeError(
        "V3 timed loop contains hashing."
    )


# There must be a post-loop fingerprint call.
post_loop_text = "\n".join(
    v3_source_lines[
        timing_loop.end_lineno:
    ]
)


if "reference_fingerprint = fingerprint(" not in post_loop_text:

    raise RuntimeError(
        "V3 lacks required post-timing integrity fingerprint."
    )


# Timer ordering itself must remain materialize-only.
timer_start_pos = timing_segment.find(
    "start_ns = time.perf_counter_ns()"
)

materialize_pos = timing_segment.find(
    "source.materialize_batch("
)

timer_stop_pos = timing_segment.find(
    "end_ns = time.perf_counter_ns()"
)


if not (
    0
    <=
    timer_start_pos
    <
    materialize_pos
    <
    timer_stop_pos
):

    raise RuntimeError(
        "V3 timer/materialization ordering changed."
    )


print(
    "\nRepeated timed loop fingerprint read:",
    "ABSENT"
)

print(
    "Post-all-timing integrity fingerprint:",
    "PRESENT"
)

print(
    "Timer boundary:",
    "UNCHANGED_MATERIALIZE_ONLY"
)


# =============================================================================
# 12. VERIFY MATERIALIZATION IMPLEMENTATION IS IDENTICAL
# =============================================================================

banner(
    "STAGE26-8B :: MATERIALIZATION IMPLEMENTATION IDENTITY"
)


v2_tree = ast.parse(
    v2_text
)


def function_source(
    tree,
    lines,
    name,
):

    matches = [
        node
        for node in ast.walk(
            tree
        )
        if (
            isinstance(
                node,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                ),
            )
            and
            node.name
            ==
            name
        )
    ]


    if len(
        matches
    ) != 1:

        raise RuntimeError(
            f"Could not uniquely resolve function {name}."
        )


    node = matches[
        0
    ]


    return "\n".join(
        lines[
            node.lineno - 1:
            node.end_lineno
        ]
    )


v2_materialize = function_source(
    v2_tree,
    v2_lines,
    "materialize_batch",
)

v3_materialize = function_source(
    v3_tree,
    v3_source_lines,
    "materialize_batch",
)


materialize_identical = (
    v2_materialize
    ==
    v3_materialize
)


print(
    "materialize_batch exact source equality:",
    materialize_identical
)


if not materialize_identical:

    raise RuntimeError(
        "V3 modified materialize_batch, which is forbidden."
    )


v2_reconstruct = function_source(
    v2_tree,
    v2_lines,
    "reconstruct",
)

v3_reconstruct = function_source(
    v3_tree,
    v3_source_lines,
    "reconstruct",
)


if v2_reconstruct != v3_reconstruct:

    raise RuntimeError(
        "V3 modified reconstruct(), which is forbidden."
    )


print(
    "reconstruct exact source equality      : True"
)


# =============================================================================
# 13. FREEZE RANDOMIZED PAIRED EXECUTION PLAN
# =============================================================================

banner(
    "STAGE26-8B :: FREEZE RANDOMIZED PAIRED EXECUTION PLAN"
)


rng = np.random.default_rng(
    RANDOMIZATION_SEED
)


if type(
    rng.bit_generator
).__name__ != "PCG64":

    raise RuntimeError(
        "default_rng is not PCG64."
    )


pair_blocks = []


for batch in BATCHES:

    for pair_rep in range(
        1,
        PAIR_REPLICATES + 1,
    ):

        if int(
            rng.integers(
                0,
                2,
            )
        ) == 0:

            within_pair_order = [
                "V2_ORIGINAL",
                "V3_CLEAN",
            ]

        else:

            within_pair_order = [
                "V3_CLEAN",
                "V2_ORIGINAL",
            ]


        pair_blocks.append(
            {
                "batch_size":
                    batch,

                "pair_rep":
                    pair_rep,

                "within_pair_order":
                    within_pair_order,
            }
        )


# Shuffle pair BLOCKS after internal order is frozen.
permutation = rng.permutation(
    len(
        pair_blocks
    )
)


pair_blocks = [
    pair_blocks[
        int(i)
    ]
    for i in permutation
]


for execution_index, block in enumerate(
    pair_blocks,
    start=1,
):

    block[
        "pair_block_execution_index"
    ] = execution_index


if len(
    pair_blocks
) != 25:

    raise RuntimeError(
        "Expected exactly 25 pair blocks."
    )


print(
    "Randomization seed:",
    RANDOMIZATION_SEED
)

print(
    "Bit generator:",
    type(
        rng.bit_generator
    ).__name__
)

print(
    "Pair blocks:",
    len(
        pair_blocks
    )
)

print(
    "Fresh worker processes:",
    len(
        pair_blocks
    )
    *
    2
)


print(
    "\nFrozen order:"
)


for block in pair_blocks:

    print(
        f"  block={block['pair_block_execution_index']:02d} "
        f"B={block['batch_size']:5d} "
        f"pair={block['pair_rep']} "
        f"order={block['within_pair_order']}"
    )


execution_plan_payload = {
    "schema":
        "stage26_8b_representation_sensitivity_execution_plan_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-8B",

    "status":
        "FROZEN_BEFORE_SENSITIVITY_TIMING",

    "scientific_parent":
        EXPECTED_PARENT,

    "randomization": {
        "rng":
            "numpy.random.default_rng_PCG64",

        "seed":
            RANDOMIZATION_SEED,

        "pair_blocks":
            25,

        "pair_replicates_per_batch":
            PAIR_REPLICATES,

        "fresh_processes":
            50,
    },

    "pair_blocks":
        pair_blocks,
}


atomic_json(
    EXECUTION_PLAN,
    execution_plan_payload,
)


execution_plan_sha = sha256_file(
    EXECUTION_PLAN
)


# =============================================================================
# 14. FREEZE SCIENTIFIC PROTOCOL
# =============================================================================

banner(
    "STAGE26-8B :: WRITE SENSITIVITY PROTOCOL"
)


protocol_payload = {
    "schema":
        "stage26_8b_representation_sensitivity_protocol_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-8B",

    "status":
        "FROZEN_BEFORE_ANY_SENSITIVITY_TIMING",

    "scientific_parent":
        EXPECTED_PARENT,

    "trigger": {
        "audit":
            "STAGE26-8A1",

        "audit_sha256":
            EXPECTED_8A1_SHA256,

        "classification":
            "OUTSIDE_TIMER_BEFORE_NEXT_TIMED_ITERATION",

        "historical_worker_sha256":
            EXPECTED_V2_SHA256,

        "issue":
            (
                "Historical V2 performs a full-output fingerprint read "
                "after every timed materialization and before the next "
                "timed iteration, creating a possible inter-iteration "
                "cache/CPU-state perturbation."
            ),
    },

    "experimental_design": {
        "type":
            "PAIRED_CONTEMPORANEOUS_A_B_SENSITIVITY",

        "A":
            "EXACT_HISTORICAL_V2_WORKER",

        "B":
            "CLEAN_V3_SENSITIVITY_WORKER",

        "same_runtime_required":
            True,

        "historical_absolute_timing_used_as_A_control":
            False,

        "reason":
            (
                "A and B must be measured contemporaneously so runtime or "
                "hardware differences from the historical Kaggle session "
                "cannot be attributed to the fingerprint change."
            ),

        "hardware_mode":
            "CPU_1_PHYSICAL_CORE",

        "batch_sizes":
            BATCHES,

        "pair_replicates_per_batch":
            PAIR_REPLICATES,

        "total_pair_blocks":
            25,

        "fresh_process_per_worker_execution":
            True,

        "total_fresh_processes":
            50,

        "execution_plan_sha256":
            execution_plan_sha,
    },

    "worker_identity": {
        "V2_original": {
            "repo_relative_path":
                str(
                    V2_WORKER.relative_to(
                        REPO
                    )
                ),

            "sha256":
                EXPECTED_V2_SHA256,

            "inter_iteration_full_output_fingerprint":
                True,
        },

        "V3_clean": {
            "repo_relative_path":
                str(
                    V3_WORKER.relative_to(
                        REPO
                    )
                ),

            "sha256":
                v3_sha,

            "inter_iteration_full_output_fingerprint":
                False,

            "post_all_timing_integrity_fingerprint":
                True,

            "materialize_batch_identical_to_V2":
                True,

            "reconstruct_identical_to_V2":
                True,

            "timer_boundary_identical_to_V2":
                True,
        },

        "V2_to_V3_diff_sha256":
            diff_sha,
    },

    "historical_condition_configs": {
        "policy":
            (
                "For every batch, preserve the committed historical 4C3 "
                "benchmark configuration semantics including start index, "
                "batch size, warmup runs and timed runs."
            ),

        "conditions":
            condition_configs,
    },

    "randomization": {
        "rng":
            "numpy.random.default_rng_PCG64",

        "seed":
            RANDOMIZATION_SEED,

        "shuffle_pair_blocks":
            True,

        "randomize_worker_order_within_pair":
            True,

        "adaptive_reordering_allowed":
            False,
    },

    "environment_policy": {
        "same_Kaggle_runtime_for_all_A_B_measurements":
            True,

        "CPU1_affinity_required":
            True,

        "GPU_allowed":
            False,

        "environment_gate": {
            "max_CPU_percent":
                20.0,

            "min_available_RAM_GiB":
                8.0,

            "attempts":
                3,

            "retry_seconds":
                5,
        },

        "pair_validity":
            (
                "A pair contributes to paired sensitivity summaries only "
                "when both its V2 and V3 worker executions PASS."
            ),

        "replacement_pair_after_failure":
            False,

        "failed_pair_policy":
            "REPORT_AS_INCOMPLETE_PAIR_WITHOUT_REPLACEMENT",
    },

    "timing_boundary": {
        "included":
            "source.materialize_batch(indices)",

        "timer":
            "time.perf_counter_ns",

        "V2_integrity_read":
            "AFTER_TIMER_INSIDE_REPEATED_TIMED_LOOP",

        "V3_integrity_read":
            "AFTER_ALL_REPEATED_TIMED_ITERATIONS",

        "integrity_read_included_in_reported_latency":
            False,
    },

    "primary_estimand": {
        "name":
            "throughput_ratio_clean_over_original",

        "per_pair_formula":
            (
                "median_throughput_V3 / median_throughput_V2"
            ),

        "statistic_source":
            "raw timed flows_per_second observations",

        "within_worker_summary":
            "np.median",

        "across_five_pairs_reporting": [
            "ALL_FIVE_PAIR_RATIOS",
            "MEDIAN_PAIR_RATIO",
            "MIN_PAIR_RATIO",
            "MAX_PAIR_RATIO",
        ],
    },

    "secondary_estimands": {
        "p50_latency_ratio_clean_over_original":
            "p50_latency_V3 / p50_latency_V2",

        "p95_latency_ratio_clean_over_original":
            "p95_latency_V3 / p95_latency_V2",

        "p99_latency_ratio_clean_over_original_when_n_gte_100":
            "p99_latency_V3 / p99_latency_V2",

        "latency_unit":
            "milliseconds",

        "latency_conversion":
            (
                "float64 elapsed_ns / 1_000_000.0 before percentile"
            ),

        "percentile_method":
            "numpy.percentile(method='linear')",
    },

    "inference_policy": {
        "hypothesis_test":
            False,

        "p_value":
            False,

        "bootstrap":
            False,

        "confidence_interval_for_pair_ratio":
            False,

        "materiality_threshold":
            None,

        "post_hoc_threshold_allowed":
            False,

        "interpretation":
            "DESCRIPTIVE_PAIRED_SENSITIVITY_ONLY",
    },

    "publication_policy": {
        "historical_stage26_4c3_remains_original_measurement":
            True,

        "historical_stage26_4c3_overwritten":
            False,

        "V3_becomes_replacement_measurement":
            False,

        "stage26_7c_recomputed":
            False,

        "stage26_7c_component_ratios_replaced":
            False,

        "sensitivity_role":
            (
                "ROBUSTNESS_EVIDENCE_FOR_INTER_ITERATION_"
                "INTEGRITY_READ_SENSITIVITY"
            ),

        "required_reporting":
            (
                "Distinguish the original Stage26-4C3 measurement from "
                "the paired V2-vs-V3 sensitivity result."
            ),
    },

    "prohibited": [
        "COMPARE_NEW_V3_DIRECTLY_TO_HISTORICAL_4C3_AS_CAUSAL_EFFECT",
        "OVERWRITE_STAGE26_4C3",
        "RECOMPUTE_STAGE26_7C_DURING_SENSITIVITY",
        "INVENT_POST_HOC_MATERIALITY_THRESHOLD",
        "BOOTSTRAP_SENSITIVITY_RESULTS",
        "HYPOTHESIS_TEST_SENSITIVITY_RESULTS",
        "MODEL_INFERENCE",
        "PARETO_RECOMPUTATION",
        "PCAP_ACCESS",
        "GPU_USE",
    ],
}


atomic_json(
    PROTOCOL,
    protocol_payload,
)


protocol_sha = sha256_file(
    PROTOCOL
)


# =============================================================================
# 15. FREEZE RECEIPT
# =============================================================================

freeze_receipt_payload = {
    "schema":
        "stage26_8b_representation_sensitivity_freeze_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-8B",

    "status":
        "PASS_PAIRED_SENSITIVITY_PROTOCOL_FROZEN_BEFORE_TIMING",

    "scientific_parent":
        EXPECTED_PARENT,

    "trigger": {
        "classification":
            "OUTSIDE_TIMER_BEFORE_NEXT_TIMED_ITERATION",

        "stage26_8a1_sha256":
            EXPECTED_8A1_SHA256,
    },

    "locked_artifacts": {
        "protocol_sha256":
            protocol_sha,

        "execution_plan_sha256":
            execution_plan_sha,

        "V2_worker_sha256":
            EXPECTED_V2_SHA256,

        "V3_worker_sha256":
            v3_sha,

        "V2_to_V3_diff_sha256":
            diff_sha,
    },

    "design": {
        "paired":
            True,

        "contemporaneous_A_B":
            True,

        "hardware":
            "CPU_1_PHYSICAL_CORE",

        "batches":
            BATCHES,

        "pair_replicates_per_batch":
            PAIR_REPLICATES,

        "pair_blocks":
            25,

        "fresh_worker_processes":
            50,

        "randomization_seed":
            RANDOMIZATION_SEED,

        "adaptive_reordering":
            False,

        "replacement_pairs":
            False,
    },

    "V3_integrity": {
        "materialize_batch_identical_to_V2":
            True,

        "reconstruct_identical_to_V2":
            True,

        "timer_boundary_identical":
            True,

        "inter_iteration_fingerprint_absent":
            True,

        "post_all_timing_integrity_fingerprint_present":
            True,
    },

    "scientific_state": {
        "corrected_or_sensitivity_timing_executed":
            False,

        "representation_executed":
            False,

        "historical_4C3_modified":
            False,

        "historical_4C3_replaced":
            False,

        "stage26_7c_recomputed":
            False,

        "stage26_7c_modified":
            False,

        "bootstrap_executed":
            False,

        "hypothesis_test_performed":
            False,

        "materiality_threshold_selected":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "Pareto_recomputed":
            False,

        "PCAP_accessed":
            False,

        "Release_corpus_accessed":
            False,

        "GPU_used":
            False,
    },

    "next_permitted_action":
        (
            "After remote Git verification, restore/verify the exact compact "
            "representation inputs and execute only the frozen paired V2/V3 "
            "Stage26-8B sensitivity plan."
        ),
}


atomic_json(
    FREEZE_RECEIPT,
    freeze_receipt_payload,
)


freeze_receipt_sha = sha256_file(
    FREEZE_RECEIPT
)


# =============================================================================
# 16. MANIFEST
# =============================================================================

package_files = [
    PROTOCOL,
    EXECUTION_PLAN,
    V3_WORKER,
    V2_TO_V3_DIFF,
    FREEZE_RECEIPT,
    DURABLE_8A,
    DURABLE_8A1,
]


manifest_rows = []


for path in package_files:

    manifest_rows.append(
        {
            "repo_relative_path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest_payload = {
    "schema":
        "stage26_8b_representation_sensitivity_lock_manifest_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-8B",

    "status":
        "READY_FOR_GIT_ANCHOR",

    "scientific_parent":
        EXPECTED_PARENT,

    "commit_subject":
        COMMIT_SUBJECT,

    "protocol_sha256":
        protocol_sha,

    "execution_plan_sha256":
        execution_plan_sha,

    "V2_worker_sha256":
        EXPECTED_V2_SHA256,

    "V3_worker_sha256":
        v3_sha,

    "V2_to_V3_diff_sha256":
        diff_sha,

    "freeze_receipt_sha256":
        freeze_receipt_sha,

    "stage26_8a_sha256":
        EXPECTED_8A_SHA256,

    "stage26_8a1_sha256":
        EXPECTED_8A1_SHA256,

    "file_count_excluding_manifest":
        len(
            manifest_rows
        ),

    "files":
        manifest_rows,

    "scientific_state": {
        "protocol_frozen":
            True,

        "execution_order_frozen":
            True,

        "V3_worker_frozen":
            True,

        "sensitivity_results_seen":
            False,

        "timing_executed":
            False,

        "historical_4C3_modified":
            False,

        "stage26_7c_modified":
            False,

        "GPU_used":
            False,
    },
}


atomic_json(
    MANIFEST,
    manifest_payload,
)


manifest_sha = sha256_file(
    MANIFEST
)


# =============================================================================
# 17. LOCAL PACKAGE AUDIT
# =============================================================================

banner(
    "STAGE26-8B :: LOCAL PACKAGE AUDIT"
)


for row in manifest_rows:

    path = (
        REPO
        /
        row[
            "repo_relative_path"
        ]
    )


    actual_size = int(
        path.stat().st_size
    )

    actual_sha = sha256_file(
        path
    )


    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Local Stage26-8B package audit failed."
        )


print(
    "\nProtocol SHA256      :",
    protocol_sha
)

print(
    "Execution plan SHA256:",
    execution_plan_sha
)

print(
    "V3 worker SHA256     :",
    v3_sha
)

print(
    "V2 -> V3 diff SHA256 :",
    diff_sha
)

print(
    "Freeze receipt SHA256:",
    freeze_receipt_sha
)

print(
    "Manifest SHA256      :",
    manifest_sha
)


# =============================================================================
# 18. GIT CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-8B :: GIT CHANGE AUDIT"
)


repo_status = git(
    "status",
    "--porcelain",
)


print(
    repo_status
)


if not repo_status:

    raise RuntimeError(
        "Expected uncommitted Stage26-8B package."
    )


unexpected = []


for line in repo_status.splitlines():

    relpath = line[
        3:
    ]


    if not relpath.startswith(
        str(
            CHECKPOINT_REL
        )
        +
        "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository changes:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 19. GIT IDENTITY
# =============================================================================

author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_PARENT,
)

author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_PARENT,
)


if (
    not author_name.strip()
    or
    "@"
    not in
    author_email
):

    raise RuntimeError(
        "Could not recover Git identity."
    )


git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


print(
    "\nGit author:"
)

print(
    " ",
    author_name,
    "<" + author_email + ">"
)


# =============================================================================
# 20. COMMIT
# =============================================================================

banner(
    "STAGE26-8B :: COMMIT"
)


git(
    "add",
    str(
        CHECKPOINT_REL
    ),
)


print(
    git(
        "diff",
        "--cached",
        "--name-status",
    )
)


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-8B commit parent mismatch."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Stage26-8B commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository not clean after Stage26-8B commit."
    )


# =============================================================================
# 21. PUSH
# =============================================================================

banner(
    "STAGE26-8B :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        / "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    push_result = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
        text=True,
    )


    print(
        push_result.stdout.strip()
    )


github_token = None


# =============================================================================
# 22. REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-8B :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "origin/main did not advance to Stage26-8B."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote Stage26-8B parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote Stage26-8B subject mismatch."
    )


# =============================================================================
# 23. REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-8B :: REMOTE BYTE VERIFICATION"
)


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    str(
        MANIFEST.relative_to(
            REPO
        )
    ),
)


remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "Remote manifest SHA256:"
)

print(
    " ",
    remote_manifest_sha
)


if remote_manifest_sha != manifest_sha:

    raise RuntimeError(
        "Remote Stage26-8B manifest mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


for row in remote_manifest[
    "files"
]:

    data = git_blob_bytes(
        "origin/main",
        row[
            "repo_relative_path"
        ],
    )


    actual_size = len(
        data
    )

    actual_sha = sha256_bytes(
        data
    )


    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    print(
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:10,d} B "
        f"{actual_sha} "
        f"{row['repo_relative_path']}"
    )


    if not passed:

        raise RuntimeError(
            "Remote Stage26-8B byte verification failed."
        )


# =============================================================================
# 24. REMOTE SCIENTIFIC VERIFICATION
# =============================================================================

banner(
    "STAGE26-8B :: REMOTE SCIENTIFIC VERIFICATION"
)


remote_protocol = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            PROTOCOL.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_plan = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            EXECUTION_PLAN.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_receipt = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            FREEZE_RECEIPT.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_v3 = git_blob_bytes(
    "origin/main",
    str(
        V3_WORKER.relative_to(
            REPO
        )
    ),
)


remote_checks = {
    "frozen before sensitivity timing":
        (
            remote_protocol[
                "status"
            ]
            ==
            "FROZEN_BEFORE_ANY_SENSITIVITY_TIMING"
        ),

    "paired contemporaneous design":
        (
            remote_protocol[
                "experimental_design"
            ][
                "type"
            ]
            ==
            "PAIRED_CONTEMPORANEOUS_A_B_SENSITIVITY"
        ),

    "five pair replicates":
        (
            remote_protocol[
                "experimental_design"
            ][
                "pair_replicates_per_batch"
            ]
            ==
            5
        ),

    "25 pair blocks":
        (
            len(
                remote_plan[
                    "pair_blocks"
                ]
            )
            ==
            25
        ),

    "50 fresh processes":
        (
            remote_protocol[
                "experimental_design"
            ][
                "total_fresh_processes"
            ]
            ==
            50
        ),

    "randomization seed 26042":
        (
            remote_plan[
                "randomization"
            ][
                "seed"
            ]
            ==
            26042
        ),

    "V2 exact historical SHA":
        (
            remote_protocol[
                "worker_identity"
            ][
                "V2_original"
            ][
                "sha256"
            ]
            ==
            EXPECTED_V2_SHA256
        ),

    "V3 exact locked SHA":
        (
            sha256_bytes(
                remote_v3
            )
            ==
            v3_sha
        ),

    "V3 materialization unchanged":
        (
            remote_protocol[
                "worker_identity"
            ][
                "V3_clean"
            ][
                "materialize_batch_identical_to_V2"
            ]
            is True
        ),

    "V3 no inter-iteration fingerprint":
        (
            remote_protocol[
                "worker_identity"
            ][
                "V3_clean"
            ][
                "inter_iteration_full_output_fingerprint"
            ]
            is False
        ),

    "V3 post-timing integrity present":
        (
            remote_protocol[
                "worker_identity"
            ][
                "V3_clean"
            ][
                "post_all_timing_integrity_fingerprint"
            ]
            is True
        ),

    "no materiality threshold":
        (
            remote_protocol[
                "inference_policy"
            ][
                "materiality_threshold"
            ]
            is None
        ),

    "no bootstrap":
        (
            remote_protocol[
                "inference_policy"
            ][
                "bootstrap"
            ]
            is False
        ),

    "historical 4C3 retained":
        (
            remote_protocol[
                "publication_policy"
            ][
                "historical_stage26_4c3_remains_original_measurement"
            ]
            is True
        ),

    "V3 not replacement":
        (
            remote_protocol[
                "publication_policy"
            ][
                "V3_becomes_replacement_measurement"
            ]
            is False
        ),

    "7C not recomputed":
        (
            remote_protocol[
                "publication_policy"
            ][
                "stage26_7c_recomputed"
            ]
            is False
        ),

    "timing not executed":
        (
            remote_receipt[
                "scientific_state"
            ][
                "corrected_or_sensitivity_timing_executed"
            ]
            is False
        ),

    "GPU false":
        (
            remote_receipt[
                "scientific_state"
            ][
                "GPU_used"
            ]
            is False
        ),
}


for name, passed in remote_checks.items():

    print(
        f"{name:42s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    remote_checks.values()
):

    raise RuntimeError(
        "Remote Stage26-8B scientific verification failed."
    )


# =============================================================================
# 25. FINAL CLOSURE
# =============================================================================

banner(
    "STAGE26-8B PAIRED REPRESENTATION SENSITIVITY FREEZE COMPLETE"
)


final_status = git(
    "status",
    "--porcelain",
)


print(
    "NEW DURABLE COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nPARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nTRIGGER:"
)

print(
    "  exact V2 worker:",
    EXPECTED_V2_SHA256
)

print(
    "  classification : OUTSIDE_TIMER_BEFORE_NEXT_TIMED_ITERATION"
)


print(
    "\nFROZEN DESIGN:"
)

print(
    "  design                 : paired contemporaneous V2 vs V3"
)

print(
    "  hardware               : CPU1"
)

print(
    "  batches                :",
    BATCHES
)

print(
    "  pair replicates/batch  :",
    PAIR_REPLICATES
)

print(
    "  pair blocks            : 25"
)

print(
    "  fresh worker processes : 50"
)

print(
    "  randomization seed     :",
    RANDOMIZATION_SEED
)

print(
    "  adaptive reorder       : NO"
)

print(
    "  replacement pairs      : NO"
)


print(
    "\nV3:"
)

print(
    "  SHA256                         :",
    v3_sha
)

print(
    "  materialize_batch changed      : NO"
)

print(
    "  reconstruct changed            : NO"
)

print(
    "  timer boundary changed         : NO"
)

print(
    "  inter-iteration fingerprint    : REMOVED"
)

print(
    "  post-all-timing integrity read : PRESENT"
)


print(
    "\nSTATISTICAL POLICY:"
)

print(
    "  primary : median throughput V3 / V2"
)

print(
    "  secondary: p50/p95 latency V3 / V2"
)

print(
    "  p99      : when n >= 100"
)

print(
    "  hypothesis test       : NO"
)

print(
    "  bootstrap             : NO"
)

print(
    "  materiality threshold : NONE"
)

print(
    "  post-hoc threshold    : FORBIDDEN"
)


print(
    "\nPUBLICATION POLICY:"
)

print(
    "  original Stage26-4C3 retained : YES"
)

print(
    "  V3 replaces 4C3               : NO"
)

print(
    "  Stage26-7C recomputed          : NO"
)

print(
    "  role                            : paired robustness evidence"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  sensitivity results seen : NO"
)

print(
    "  sensitivity timing run   : NO"
)

print(
    "  representation run       : NO"
)

print(
    "  historical 4C3 modified  : NO"
)

print(
    "  Stage26-7C modified      : NO"
)

print(
    "  model inference          : NO"
)

print(
    "  bootstrap                : NO"
)

print(
    "  Pareto                   : NO"
)

print(
    "  GPU                      : NO"
)


print(
    "\nHASHES:"
)

print(
    "  protocol      :",
    protocol_sha
)

print(
    "  execution plan:",
    execution_plan_sha
)

print(
    "  V2 worker     :",
    EXPECTED_V2_SHA256
)

print(
    "  V3 worker     :",
    v3_sha
)

print(
    "  V2->V3 diff   :",
    diff_sha
)

print(
    "  freeze receipt:",
    freeze_receipt_sha
)

print(
    "  manifest      :",
    manifest_sha
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  commit             : PASS"
)

print(
    "  parent             : PASS"
)

print(
    "  package bytes      : PASS"
)

print(
    "  paired design      : PASS"
)

print(
    "  V3 implementation  : PASS"
)

print(
    "  scientific policy  : PASS"
)


print(
    "\nRepo clean:",
    final_status == ""
)


if final_status:

    raise RuntimeError(
        "Repository not clean after Stage26-8B."
    )


print(
    "\nNEXT:"
)

print(
    "  Restore and hash-verify the exact compact representation inputs."
)

print(
    "  Then execute ONLY the remotely anchored 25-pair / 50-process"
)

print(
    "  V2-vs-V3 sensitivity plan."
)

print(
    "  Historical Stage26-4C3 remains untouched."
)

print(
    "  GPU remains OFF."
)


STAGE26-8B :: DURABLE SCIENTIFIC PARENT
Expected parent: 9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3
Local HEAD     : 9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3
origin/main    : 9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3
Repo clean     : True

STAGE26-8B :: EXACT SOURCE / PROVENANCE IDENTITY
8A transient audit           PASS 0d457c476567426c0093d29222450f411e94732455139a5d2f39ad2c0d1ed73c
8A1 exact-worker audit       PASS e2121362d23a8e81afd367d42df944f8558422c0984762c5fbca5dd5f186a9c0
historical worker V2         PASS ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23
4C3 summary CSV              PASS ca5bbc17edfa4bb088599236b4099c1986fd73cb7f087ef3ac753a524afd2ba5
4C3 summary JSON             PASS 63c8041144668c62218fc9c46eb6a57aba9dcda0c9a3f0477f932f6a572c34f7
4C3 receipt                  PASS ed55b65905080dd6b796a54fb720de76bf100fdc7f3271a973e49c81f1722b14
4C3 manifest                 PASS bfbc9000a708a803c53a11b591e8021d54325d1b7bccebb4b0aad65e1439b9db
4C3 execution plan  

In [6]:
# =============================================================================
# STAGE26-8C0
# RESTORE + HASH-VERIFY EXACT LABEL-FREE COMPACT REPRESENTATION INPUTS
#
# DURABLE SCIENTIFIC PARENT:
#   973e0c479ce91ad21e92ed5db11585f26a4c49ec
#
# PURPOSE
# -------
# Prepare the exact representation inputs required by the frozen Stage26-8B
# paired V2/V3 sensitivity experiment.
#
# SOURCE RELEASE:
#
#   stage20-Monday-compact-corpus-v1.tar
#
# Expected TAR:
#   size:
#       595,261,440 bytes
#
#   SHA256:
#       4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20
#
# RESTORE ONLY:
#
#   encoded_bytes.bin
#   flow_offsets.npy
#   packet_lengths.npy
#
# DO NOT RESTORE:
#
#   labels.npy
#
# Exact target runtime directory:
#
#   /kaggle/working/stage26_deployment_profiling/
#       representation/stage26_4c2_equivalence/
#       Monday_release_representation_subset
#
# This is the corpus_dir frozen in the historical Stage26-4C3 configs.
#
# THIS CELL DOES NOT:
#   - run V2
#   - run V3
#   - perform representation timing
#   - calculate sensitivity results
#   - load models
#   - perform inference
#   - bootstrap
#   - recompute Stage26-7
#   - recompute Pareto
#   - access PCAP
#   - extract labels
#   - use GPU
#   - modify Git
# =============================================================================

from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
from pathlib import Path

import numpy as np


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "973e0c479ce91ad21e92ed5db11585f26a4c49ec"
)


# -----------------------------------------------------------------------------
# Stage26-8B exact lock identities
# -----------------------------------------------------------------------------

LOCK_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_8b_representation_sensitivity_protocol_lock"
)

PROTOCOL = (
    LOCK_DIR
    / "stage26_8b_representation_sensitivity_protocol.json"
)

EXECUTION_PLAN = (
    LOCK_DIR
    / "stage26_8b_representation_sensitivity_execution_plan.json"
)

V3_WORKER = (
    LOCK_DIR
    / "stage26_representation_worker_v3_sensitivity.py"
)

FREEZE_RECEIPT = (
    LOCK_DIR
    / "stage26_8b_representation_sensitivity_freeze_receipt.json"
)

LOCK_MANIFEST = (
    LOCK_DIR
    / "stage26_8b_representation_sensitivity_lock_manifest.json"
)


EXPECTED_PROTOCOL_SHA256 = (
    "4ce96c1540b29bd9683305ed981412b46be4b0b2596dcb58b3064f0724f05c5e"
)

EXPECTED_EXECUTION_PLAN_SHA256 = (
    "5581bb2576919930913f49a308470083e7b27e315bc5cd99ba1fc1a853dcac14"
)

EXPECTED_V3_SHA256 = (
    "c065e70fc16a628e482961f0db669d95affb841edd39153abf28dc99e868ad02"
)

EXPECTED_FREEZE_RECEIPT_SHA256 = (
    "d052bcd8bf3b2ed6a93f1ca3813036c030fa1d1de1fb28ffea166cf3e226bc33"
)

EXPECTED_LOCK_MANIFEST_SHA256 = (
    "6cfa4790ccabd1d755af40960b2e38846b4e15fe92ba2fac70d72ac3c162334f"
)


# -----------------------------------------------------------------------------
# Historical V2
# -----------------------------------------------------------------------------

V2_WORKER = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c1a_label_free_equivalence_erratum"
    / "stage26_representation_worker_v2.py"
)

EXPECTED_V2_SHA256 = (
    "ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23"
)


# -----------------------------------------------------------------------------
# Exact compact release
# -----------------------------------------------------------------------------

RELEASE_FILENAME = (
    "stage20-Monday-compact-corpus-v1.tar"
)

EXPECTED_RELEASE_SIZE = (
    595_261_440
)

EXPECTED_RELEASE_SHA256 = (
    "4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20"
)


EXPECTED_FILES = {
    "encoded_bytes.bin": {
        "size_bytes":
            522_845_159,

        "sha256":
            "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",
    },

    "flow_offsets.npy": {
        "size_bytes":
            4_228_208,

        "sha256":
            "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",
    },

    "packet_lengths.npy": {
        "size_bytes":
            67_649_280,

        "sha256":
            "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
    },
}


FORBIDDEN_FILE = (
    "labels.npy"
)

EXPECTED_FLOW_COUNT = (
    528_509
)


# Exact path expected by historical configs.
RESTORE_DIR = Path(
    "/kaggle/working/"
    "stage26_deployment_profiling/"
    "representation/"
    "stage26_4c2_equivalence/"
    "Monday_release_representation_subset"
)


# Transient restoration receipt — outside Git.
RUNTIME_RECEIPT = Path(
    "/kaggle/working/"
    "stage26_deployment_profiling/"
    "cpu_closure/"
    "stage26_8c0_compact_input_restore_receipt.json"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            p.stdout
        )

    return p


def git(*args):

    return run(
        [
            "git",
            *args,
        ]
    ).stdout.strip()


def sha256_file(
    path,
    *,
    progress=False,
):

    path = Path(path)

    h = hashlib.sha256()

    total = 0

    next_report = (
        512 * 1024 * 1024
    )


    with path.open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

            total += len(
                block
            )


            if (
                progress
                and
                total >= next_report
            ):

                print(
                    f"  hashed {total / (1024**3):.2f} GiB"
                )

                next_report += (
                    512 * 1024 * 1024
                )


    return h.hexdigest()


def atomic_json(
    path,
    payload,
):

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path)
        +
        ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )


    os.replace(
        tmp,
        path,
    )


# =============================================================================
# 2. DURABLE GIT STATE
# =============================================================================

banner(
    "STAGE26-8C0 :: DURABLE SCIENTIFIC STATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-8C0 local HEAD."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Stage26-8C0."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if RUNTIME_RECEIPT.exists():

    raise RuntimeError(
        "Stage26-8C0 transient receipt already exists."
    )


# =============================================================================
# 3. EXACT STAGE26-8B LOCK GATE
# =============================================================================

banner(
    "STAGE26-8C0 :: EXACT STAGE26-8B LOCK IDENTITY"
)


lock_checks = [
    (
        "protocol",
        PROTOCOL,
        EXPECTED_PROTOCOL_SHA256,
    ),

    (
        "execution plan",
        EXECUTION_PLAN,
        EXPECTED_EXECUTION_PLAN_SHA256,
    ),

    (
        "V2 worker",
        V2_WORKER,
        EXPECTED_V2_SHA256,
    ),

    (
        "V3 worker",
        V3_WORKER,
        EXPECTED_V3_SHA256,
    ),

    (
        "freeze receipt",
        FREEZE_RECEIPT,
        EXPECTED_FREEZE_RECEIPT_SHA256,
    ),

    (
        "lock manifest",
        LOCK_MANIFEST,
        EXPECTED_LOCK_MANIFEST_SHA256,
    ),
]


for label, path, expected in lock_checks:

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:18s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Stage26-8B identity mismatch: {label}"
        )


protocol = json.loads(
    PROTOCOL.read_text(
        encoding="utf-8"
    )
)


if protocol[
    "status"
] != "FROZEN_BEFORE_ANY_SENSITIVITY_TIMING":

    raise RuntimeError(
        "Stage26-8B protocol not frozen before timing."
    )


if protocol[
    "experimental_design"
][
    "total_pair_blocks"
] != 25:

    raise RuntimeError(
        "Frozen pair-block count changed."
    )


if protocol[
    "experimental_design"
][
    "total_fresh_processes"
] != 50:

    raise RuntimeError(
        "Frozen worker-process count changed."
    )


print(
    "\nStage26-8B protocol gate: PASS"
)


# =============================================================================
# 4. DISCOVER EXACT RELEASE TAR
# =============================================================================

banner(
    "STAGE26-8C0 :: DISCOVER EXACT COMPACT RELEASE"
)


INPUT_ROOT = Path(
    "/kaggle/input"
)


if not INPUT_ROOT.is_dir():

    raise RuntimeError(
        "/kaggle/input is unavailable."
    )


release_candidates = sorted(
    INPUT_ROOT.rglob(
        RELEASE_FILENAME
    )
)


print(
    "Candidates by exact filename:",
    len(
        release_candidates
    )
)


for path in release_candidates:

    print(
        " ",
        path
    )


if not release_candidates:

    # Give useful diagnostics without guessing another scientific source.
    dataset_dirs = sorted(
        path
        for path in INPUT_ROOT.iterdir()
        if path.is_dir()
    )


    print(
        "\nAttached Kaggle input directories:"
    )


    for path in dataset_dirs[
        :100
    ]:

        print(
            " ",
            path
        )


    raise FileNotFoundError(
        f"Could not find {RELEASE_FILENAME} under /kaggle/input."
    )


# Select only a candidate with BOTH exact size and exact SHA.
matching_release = []


for path in release_candidates:

    size = int(
        path.stat().st_size
    )


    print(
        f"\nChecking candidate:\n  {path}"
    )

    print(
        "  size expected:",
        EXPECTED_RELEASE_SIZE
    )

    print(
        "  size actual  :",
        size
    )


    if size != EXPECTED_RELEASE_SIZE:

        print(
            "  size gate    : FAIL"
        )

        continue


    print(
        "  computing SHA256..."
    )


    digest = sha256_file(
        path,
        progress=True,
    )


    print(
        "  SHA expected :",
        EXPECTED_RELEASE_SHA256
    )

    print(
        "  SHA actual   :",
        digest
    )


    if digest == EXPECTED_RELEASE_SHA256:

        print(
            "  identity     : PASS"
        )

        matching_release.append(
            path
        )

    else:

        print(
            "  identity     : FAIL"
        )


if len(
    matching_release
) != 1:

    raise RuntimeError(
        "Expected exactly one byte-identical compact release; "
        f"found {len(matching_release)}."
    )


release_tar = matching_release[
    0
]


print(
    "\nResolved exact release:"
)

print(
    " ",
    release_tar
)


# =============================================================================
# 5. INSPECT TAR — NO EXTRACTION YET
# =============================================================================

banner(
    "STAGE26-8C0 :: TAR MEMBER AUDIT"
)


with tarfile.open(
    release_tar,
    mode="r",
) as tf:

    members = tf.getmembers()


basename_index = {}


for member in members:

    if not member.isfile():

        continue


    basename = Path(
        member.name
    ).name


    basename_index.setdefault(
        basename,
        []
    ).append(
        member
    )


for filename in [
    *EXPECTED_FILES.keys(),
    FORBIDDEN_FILE,
]:

    matches = basename_index.get(
        filename,
        []
    )


    print(
        f"{filename:24s}: "
        f"{len(matches)} member(s)"
    )


    for member in matches:

        print(
            f"  {member.name} "
            f"({member.size:,} B)"
        )


for filename in EXPECTED_FILES:

    matches = basename_index.get(
        filename,
        []
    )


    if len(
        matches
    ) != 1:

        raise RuntimeError(
            f"Expected exactly one TAR member named {filename}; "
            f"found {len(matches)}."
        )


# labels.npy may exist in the release TAR, but MUST NOT be restored.
if len(
    basename_index.get(
        FORBIDDEN_FILE,
        []
    )
) > 1:

    raise RuntimeError(
        "Unexpected duplicate labels.npy members."
    )


print(
    "\nTAR structure gate: PASS"
)

print(
    "labels.npy extraction permitted: NO"
)


# =============================================================================
# 6. HANDLE EXISTING RESTORE DIRECTORY CONSERVATIVELY
# =============================================================================

banner(
    "STAGE26-8C0 :: RESTORE TARGET STATE"
)


print(
    "Target:"
)

print(
    " ",
    RESTORE_DIR
)

print(
    "Already exists:",
    RESTORE_DIR.exists()
)


if RESTORE_DIR.exists():

    # Never silently overwrite an existing scientific runtime input.
    existing_names = sorted(
        path.name
        for path in RESTORE_DIR.iterdir()
        if path.is_file()
    )


    print(
        "Existing files:"
    )


    for name in existing_names:

        print(
            " ",
            name
        )


    unexpected_existing = [
        name
        for name in existing_names
        if name not in EXPECTED_FILES
    ]


    if unexpected_existing:

        raise RuntimeError(
            "Existing restore directory contains unexpected files: "
            +
            repr(
                unexpected_existing
            )
        )


    if FORBIDDEN_FILE in existing_names:

        raise RuntimeError(
            "labels.npy exists in the label-free restore directory."
        )


    for filename, expected in EXPECTED_FILES.items():

        path = (
            RESTORE_DIR
            /
            filename
        )


        if not path.is_file():

            raise RuntimeError(
                f"Existing restore directory is incomplete: missing {filename}."
            )


        actual_size = int(
            path.stat().st_size
        )

        actual_sha = sha256_file(
            path
        )


        if (
            actual_size
            !=
            expected[
                "size_bytes"
            ]
            or
            actual_sha
            !=
            expected[
                "sha256"
            ]
        ):

            raise RuntimeError(
                f"Existing restored file does not match frozen identity: "
                f"{filename}"
            )


    restore_action = (
        "REUSED_EXISTING_BYTE_IDENTICAL_LABEL_FREE_SUBSET"
    )


else:

    restore_action = (
        "EXTRACTED_FROM_EXACT_RELEASE_TAR"
    )


    RESTORE_DIR.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temp_parent = RESTORE_DIR.parent


    with tempfile.TemporaryDirectory(
        prefix="stage26_8c0_restore_",
        dir=temp_parent,
    ) as temp_name:

        temp_dir = Path(
            temp_name
        )


        with tarfile.open(
            release_tar,
            mode="r",
        ) as tf:

            for filename, expected in EXPECTED_FILES.items():

                member = basename_index[
                    filename
                ][
                    0
                ]


                print(
                    f"Extracting {filename}..."
                )


                source = tf.extractfile(
                    member
                )


                if source is None:

                    raise RuntimeError(
                        f"Could not open TAR member: {member.name}"
                    )


                destination = (
                    temp_dir
                    /
                    filename
                )


                with source, destination.open(
                    "wb"
                ) as out:

                    shutil.copyfileobj(
                        source,
                        out,
                        length=8 * 1024 * 1024,
                    )


                    out.flush()

                    os.fsync(
                        out.fileno()
                    )


                actual_size = int(
                    destination.stat().st_size
                )

                actual_sha = sha256_file(
                    destination
                )


                print(
                    "  size:",
                    actual_size
                )

                print(
                    "  SHA :",
                    actual_sha
                )


                if actual_size != expected[
                    "size_bytes"
                ]:

                    raise RuntimeError(
                        f"{filename}: extracted size mismatch."
                    )


                if actual_sha != expected[
                    "sha256"
                ]:

                    raise RuntimeError(
                        f"{filename}: extracted SHA256 mismatch."
                    )


        # Explicitly prove labels were not extracted.
        if (
            temp_dir
            /
            FORBIDDEN_FILE
        ).exists():

            raise RuntimeError(
                "Forbidden labels.npy was extracted."
            )


        # Move only after every file passed.
        os.replace(
            temp_dir,
            RESTORE_DIR,
        )


# =============================================================================
# 7. FINAL BYTE-IDENTITY AUDIT
# =============================================================================

banner(
    "STAGE26-8C0 :: FINAL RESTORED INPUT IDENTITY"
)


restored_identity = {}


for filename, expected in EXPECTED_FILES.items():

    path = (
        RESTORE_DIR
        /
        filename
    )


    if not path.is_file():

        raise FileNotFoundError(
            path
        )


    actual_size = int(
        path.stat().st_size
    )

    actual_sha = sha256_file(
        path
    )


    passed = (
        actual_size
        ==
        expected[
            "size_bytes"
        ]
        and
        actual_sha
        ==
        expected[
            "sha256"
        ]
    )


    print(
        f"{filename:24s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_size:12,d} B "
        f"{actual_sha}"
    )


    if not passed:

        raise RuntimeError(
            f"Final restored identity mismatch: {filename}"
        )


    restored_identity[
        filename
    ] = {
        "size_bytes":
            actual_size,

        "sha256":
            actual_sha,
    }


labels_path = (
    RESTORE_DIR
    /
    FORBIDDEN_FILE
)


print(
    "\nlabels.npy present:",
    labels_path.exists()
)


if labels_path.exists():

    raise RuntimeError(
        "Label-free Stage26-8 sensitivity input contains labels.npy."
    )


# =============================================================================
# 8. STRUCTURAL / GEOMETRY AUDIT
# =============================================================================

banner(
    "STAGE26-8C0 :: LABEL-FREE STRUCTURAL GEOMETRY"
)


encoded_path = (
    RESTORE_DIR
    /
    "encoded_bytes.bin"
)

offsets_path = (
    RESTORE_DIR
    /
    "flow_offsets.npy"
)

lengths_path = (
    RESTORE_DIR
    /
    "packet_lengths.npy"
)


offsets = np.load(
    offsets_path,
    mmap_mode="r",
)

packet_lengths = np.load(
    lengths_path,
    mmap_mode="r",
)


print(
    "flow_offsets dtype :",
    offsets.dtype
)

print(
    "flow_offsets shape :",
    offsets.shape
)

print(
    "packet_lengths dtype:",
    packet_lengths.dtype
)

print(
    "packet_lengths shape:",
    packet_lengths.shape
)

print(
    "encoded byte count :",
    encoded_path.stat().st_size
)


if offsets.ndim != 1:

    raise RuntimeError(
        "flow_offsets.npy is not one-dimensional."
    )


if len(
    offsets
) != (
    EXPECTED_FLOW_COUNT
    +
    1
):

    raise RuntimeError(
        "flow_offsets length does not equal flow_count + 1."
    )


if int(
    offsets[
        0
    ]
) != 0:

    raise RuntimeError(
        "First flow offset is not zero."
    )


if int(
    offsets[
        -1
    ]
) != int(
    encoded_path.stat().st_size
):

    raise RuntimeError(
        "Final flow offset does not equal encoded byte count."
    )


if packet_lengths.shape[
    0
] != EXPECTED_FLOW_COUNT:

    raise RuntimeError(
        "packet_lengths first dimension does not equal frozen flow count."
    )


print(
    "\nFrozen flow count:",
    EXPECTED_FLOW_COUNT
)

print(
    "Geometry audit   : PASS"
)


# =============================================================================
# 9. VERIFY EVERY HISTORICAL CONFIG POINTS TO THIS EXACT DIRECTORY
# =============================================================================

banner(
    "STAGE26-8C0 :: HISTORICAL CONFIG PATH COMPATIBILITY"
)


for batch in [
    1,
    64,
    256,
    1024,
    8192,
]:

    configs = sorted(
        C4_DIR.glob(
            f"conditions/batch_{batch}/*_config.json"
        )
    )


    if len(
        configs
    ) != 1:

        raise RuntimeError(
            f"B={batch}: historical config resolution failure."
        )


    cfg = json.loads(
        configs[
            0
        ].read_text(
            encoding="utf-8"
        )
    )


    frozen_corpus_dir = Path(
        cfg[
            "corpus_dir"
        ]
    )


    same_path = (
        frozen_corpus_dir
        ==
        RESTORE_DIR
    )


    print(
        f"B={batch:5d} "
        f"corpus_path_match={'PASS' if same_path else 'FAIL'}"
    )


    if not same_path:

        raise RuntimeError(
            f"B={batch}: restored directory does not match frozen corpus_dir."
        )


# =============================================================================
# 10. WRITE TRANSIENT RESTORE RECEIPT
# =============================================================================

banner(
    "STAGE26-8C0 :: WRITE TRANSIENT RESTORE RECEIPT"
)


receipt = {
    "schema":
        "stage26_8c0_compact_input_restore_receipt_v1",

    "scientific_parent":
        EXPECTED_PARENT,

    "status":
        "PASS_EXACT_LABEL_FREE_COMPACT_INPUT_RESTORE",

    "restore_action":
        restore_action,

    "source_release": {
        "path":
            str(
                release_tar
            ),

        "filename":
            RELEASE_FILENAME,

        "size_bytes":
            int(
                release_tar.stat().st_size
            ),

        "sha256":
            EXPECTED_RELEASE_SHA256,
    },

    "restore_directory":
        str(
            RESTORE_DIR
        ),

    "restored_files":
        restored_identity,

    "forbidden_files": {
        "labels.npy_present":
            False,
    },

    "geometry": {
        "expected_flow_count":
            EXPECTED_FLOW_COUNT,

        "flow_offsets_shape":
            [
                int(x)
                for x in offsets.shape
            ],

        "flow_offsets_dtype":
            str(
                offsets.dtype
            ),

        "packet_lengths_shape":
            [
                int(x)
                for x in packet_lengths.shape
            ],

        "packet_lengths_dtype":
            str(
                packet_lengths.dtype
            ),

        "encoded_bytes":
            int(
                encoded_path.stat().st_size
            ),

        "first_offset":
            int(
                offsets[
                    0
                ]
            ),

        "final_offset":
            int(
                offsets[
                    -1
                ]
            ),
    },

    "stage26_8b_identity": {
        "protocol_sha256":
            EXPECTED_PROTOCOL_SHA256,

        "execution_plan_sha256":
            EXPECTED_EXECUTION_PLAN_SHA256,

        "V2_worker_sha256":
            EXPECTED_V2_SHA256,

        "V3_worker_sha256":
            EXPECTED_V3_SHA256,

        "freeze_receipt_sha256":
            EXPECTED_FREEZE_RECEIPT_SHA256,

        "lock_manifest_sha256":
            EXPECTED_LOCK_MANIFEST_SHA256,
    },

    "scientific_state": {
        "compact_inputs_restored":
            True,

        "labels_restored":
            False,

        "representation_executed":
            False,

        "sensitivity_timing_executed":
            False,

        "sensitivity_results_seen":
            False,

        "historical_4C3_modified":
            False,

        "stage26_7c_modified":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "bootstrap_executed":
            False,

        "Pareto_recomputed":
            False,

        "PCAP_accessed":
            False,

        "GPU_used":
            False,

        "Git_modified":
            False,
    },

    "next":
        (
            "Execute only the remotely anchored Stage26-8B "
            "25-pair / 50-process contemporaneous V2-vs-V3 "
            "representation sensitivity plan."
        ),
}


atomic_json(
    RUNTIME_RECEIPT,
    receipt,
)


runtime_receipt_sha = sha256_file(
    RUNTIME_RECEIPT
)


print(
    "Receipt:"
)

print(
    " ",
    RUNTIME_RECEIPT
)

print(
    "SHA256:"
)

print(
    " ",
    runtime_receipt_sha
)


# =============================================================================
# 11. FINAL GIT / SCIENTIFIC CLOSURE
# =============================================================================

banner(
    "STAGE26-8C0 COMPACT INPUT RESTORE COMPLETE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed during Stage26-8C0."
    )


if final_remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed during Stage26-8C0."
    )


if final_status:

    raise RuntimeError(
        "Stage26-8C0 unexpectedly modified Git."
    )


print(
    "\nRESTORE:"
)

print(
    "  source release SHA verified : YES"
)

print(
    "  encoded_bytes.bin           : VERIFIED"
)

print(
    "  flow_offsets.npy            : VERIFIED"
)

print(
    "  packet_lengths.npy          : VERIFIED"
)

print(
    "  labels.npy                  : ABSENT"
)

print(
    "  flow geometry               : VERIFIED"
)

print(
    "  historical corpus_dir       : EXACT MATCH"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  sensitivity timing run : NO"
)

print(
    "  V2 executed            : NO"
)

print(
    "  V3 executed            : NO"
)

print(
    "  results seen           : NO"
)

print(
    "  historical 4C3 changed : NO"
)

print(
    "  Stage26-7C changed     : NO"
)

print(
    "  model inference        : NO"
)

print(
    "  bootstrap              : NO"
)

print(
    "  Pareto                 : NO"
)

print(
    "  PCAP                   : NO"
)

print(
    "  GPU                    : NO"
)

print(
    "  Git modified           : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Execute the frozen Stage26-8B paired V2/V3 sensitivity plan:"
)

print(
    "    25 randomized pair blocks"
)

print(
    "    50 fresh worker processes"
)

print(
    "    CPU1 only"
)

print(
    "    no adaptive replacement"
)

print(
    "    no bootstrap"
)

print(
    "    no GPU"
)


STAGE26-8C0 :: DURABLE SCIENTIFIC STATE
Expected parent: 973e0c479ce91ad21e92ed5db11585f26a4c49ec
Local HEAD     : 973e0c479ce91ad21e92ed5db11585f26a4c49ec
origin/main    : 973e0c479ce91ad21e92ed5db11585f26a4c49ec
Repo clean     : True

STAGE26-8C0 :: EXACT STAGE26-8B LOCK IDENTITY
protocol           PASS 4ce96c1540b29bd9683305ed981412b46be4b0b2596dcb58b3064f0724f05c5e
execution plan     PASS 5581bb2576919930913f49a308470083e7b27e315bc5cd99ba1fc1a853dcac14
V2 worker          PASS ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23
V3 worker          PASS c065e70fc16a628e482961f0db669d95affb841edd39153abf28dc99e868ad02
freeze receipt     PASS d052bcd8bf3b2ed6a93f1ca3813036c030fa1d1de1fb28ffea166cf3e226bc33
lock manifest      PASS 6cfa4790ccabd1d755af40960b2e38846b4e15fe92ba2fac70d72ac3c162334f

Stage26-8B protocol gate: PASS

STAGE26-8C0 :: DISCOVER EXACT COMPACT RELEASE
Candidates by exact filename: 0

Attached Kaggle input directories:
  /kaggle/input/datasets


FileNotFoundError: Could not find stage20-Monday-compact-corpus-v1.tar under /kaggle/input.

In [7]:
# =============================================================================
# STAGE26-8C0-DIAG
# NARROW READ-ONLY KAGGLE INPUT-MOUNT DIAGNOSTIC
#
# PURPOSE
# -------
# Determine where the frozen Stage20 compact representation source exists in
# the current fresh Kaggle runtime.
#
# We search for:
#
#   1. the exact release filename;
#   2. any TAR/ZIP-like archive that may contain/represent the release;
#   3. exact required member basenames:
#        encoded_bytes.bin
#        flow_offsets.npy
#        packet_lengths.npy
#   4. any file whose byte size exactly matches the frozen release/member size.
#
# Exact SHA256 is calculated ONLY for plausible candidates.
#
# NO:
#   - extraction
#   - file copying
#   - representation execution
#   - timing
#   - V2/V3 execution
#   - bootstrap
#   - model loading
#   - inference
#   - PCAP
#   - GPU
#   - Git modification
# =============================================================================

from __future__ import annotations

import hashlib
import os
import subprocess
from collections import defaultdict
from pathlib import Path


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "973e0c479ce91ad21e92ed5db11585f26a4c49ec"
)

INPUT_ROOT = Path(
    "/kaggle/input"
)

DATASETS_ROOT = Path(
    "/kaggle/input/datasets"
)


EXPECTED = {
    "stage20-Monday-compact-corpus-v1.tar": {
        "size_bytes":
            595_261_440,

        "sha256":
            "4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20",
    },

    "encoded_bytes.bin": {
        "size_bytes":
            522_845_159,

        "sha256":
            "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",
    },

    "flow_offsets.npy": {
        "size_bytes":
            4_228_208,

        "sha256":
            "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",
    },

    "packet_lengths.npy": {
        "size_bytes":
            67_649_280,

        "sha256":
            "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
    },
}

FORBIDDEN_BASENAME = (
    "labels.npy"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def git(*args):

    p = subprocess.run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(
    path,
    *,
    progress=False,
):

    path = Path(path)

    h = hashlib.sha256()

    total = 0

    next_report = (
        512 * 1024 * 1024
    )


    with path.open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

            total += len(
                block
            )


            if (
                progress
                and
                total >= next_report
            ):

                print(
                    f"      hashed {total / (1024**3):.2f} GiB"
                )

                next_report += (
                    512 * 1024 * 1024
                )


    return h.hexdigest()


# =============================================================================
# 2. DURABLE STATE
# =============================================================================

banner(
    "STAGE26-8C0-DIAG :: DURABLE STATE"
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected local HEAD."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected origin/main."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


# =============================================================================
# 3. INPUT ROOT INVENTORY
# =============================================================================

banner(
    "STAGE26-8C0-DIAG :: INPUT ROOT INVENTORY"
)


print(
    "/kaggle/input exists:",
    INPUT_ROOT.is_dir()
)

print(
    "/kaggle/input/datasets exists:",
    DATASETS_ROOT.is_dir()
)


if not INPUT_ROOT.is_dir():

    raise RuntimeError(
        "/kaggle/input is unavailable."
    )


print(
    "\nImmediate /kaggle/input entries:"
)


for path in sorted(
    INPUT_ROOT.iterdir()
):

    print(
        " ",
        path,
        "DIR"
        if path.is_dir()
        else
        "FILE",
    )


# =============================================================================
# 4. COMPLETE RECURSIVE FILE INVENTORY — METADATA ONLY
# =============================================================================

banner(
    "STAGE26-8C0-DIAG :: RECURSIVE FILE INVENTORY"
)


all_files = []


for root, dirs, files in os.walk(
    INPUT_ROOT,
    followlinks=True,
):

    root_path = Path(
        root
    )


    for filename in files:

        path = (
            root_path
            /
            filename
        )


        try:

            size = int(
                path.stat().st_size
            )

        except OSError as exc:

            print(
                "STAT FAIL:",
                path,
                repr(
                    exc
                ),
            )

            continue


        all_files.append(
            {
                "path":
                    path,

                "basename":
                    path.name,

                "size_bytes":
                    size,

                "suffix":
                    path.suffix.lower(),
            }
        )


print(
    "Total mounted files:",
    len(
        all_files
    )
)


if not all_files:

    print(
        "\nWARNING: /kaggle/input contains no recursively visible files."
    )


# =============================================================================
# 5. EXACT BASENAME SEARCH
# =============================================================================

banner(
    "STAGE26-8C0-DIAG :: EXACT BASENAME SEARCH"
)


basename_matches = defaultdict(
    list
)


for record in all_files:

    if (
        record[
            "basename"
        ]
        in
        EXPECTED
        or
        record[
            "basename"
        ]
        ==
        FORBIDDEN_BASENAME
    ):

        basename_matches[
            record[
                "basename"
            ]
        ].append(
            record
        )


for basename in [
    *EXPECTED.keys(),
    FORBIDDEN_BASENAME,
]:

    matches = basename_matches.get(
        basename,
        []
    )


    print(
        f"\n{basename}: {len(matches)} match(es)"
    )


    for record in matches:

        print(
            f"  {record['size_bytes']:12,d} B "
            f"{record['path']}"
        )


# =============================================================================
# 6. EXACT SIZE SEARCH
# =============================================================================

banner(
    "STAGE26-8C0-DIAG :: EXACT FROZEN SIZE SEARCH"
)


expected_size_to_names = defaultdict(
    list
)


for name, spec in EXPECTED.items():

    expected_size_to_names[
        int(
            spec[
                "size_bytes"
            ]
        )
    ].append(
        name
    )


size_candidates = []


for record in all_files:

    if record[
        "size_bytes"
    ] in expected_size_to_names:

        size_candidates.append(
            record
        )


print(
    "Files matching any frozen byte size:",
    len(
        size_candidates
    )
)


for record in size_candidates:

    expected_names = expected_size_to_names[
        record[
            "size_bytes"
        ]
    ]


    print(
        f"\n  {record['size_bytes']:12,d} B "
        f"{record['path']}"
    )

    print(
        "    size corresponds to:",
        expected_names
    )


# =============================================================================
# 7. ARCHIVE-LIKE FILE INVENTORY
# =============================================================================

banner(
    "STAGE26-8C0-DIAG :: ARCHIVE-LIKE FILES"
)


archive_suffixes = {
    ".tar",
    ".zip",
    ".tgz",
    ".gz",
    ".bz2",
    ".xz",
    ".7z",
}


archive_candidates = [
    record
    for record in all_files
    if (
        record[
            "suffix"
        ]
        in
        archive_suffixes
        or
        ".tar."
        in
        record[
            "basename"
        ].lower()
    )
]


print(
    "Archive-like files:",
    len(
        archive_candidates
    )
)


for record in archive_candidates[
    :200
]:

    print(
        f"  {record['size_bytes']:12,d} B "
        f"{record['path']}"
    )


# =============================================================================
# 8. COMPACT/STAGE20/MONDAY NAME SEARCH
# =============================================================================

banner(
    "STAGE26-8C0-DIAG :: SEMANTIC FILENAME SEARCH"
)


semantic_tokens = [
    "stage20",
    "monday",
    "compact",
    "corpus",
    "encoded",
    "offset",
    "packet",
]


semantic_candidates = []


for record in all_files:

    lower = str(
        record[
            "path"
        ]
    ).lower()


    hits = [
        token
        for token in semantic_tokens
        if token in lower
    ]


    if hits:

        semantic_candidates.append(
            {
                **record,
                "hits":
                    hits,
            }
        )


print(
    "Semantic filename/path candidates:",
    len(
        semantic_candidates
    )
)


for record in semantic_candidates[
    :300
]:

    print(
        f"  {record['size_bytes']:12,d} B "
        f"hits={record['hits']} "
        f"{record['path']}"
    )


# =============================================================================
# 9. HASH PLAUSIBLE CANDIDATES ONLY
# =============================================================================

banner(
    "STAGE26-8C0-DIAG :: SHA256 OF PLAUSIBLE CANDIDATES"
)


# Candidate set:
#   - exact expected basename
#   - exact expected frozen size
#
# This also catches a byte-identical release/member that Kaggle renamed.

plausible_paths = {}


for records in basename_matches.values():

    for record in records:

        plausible_paths[
            str(
                record[
                    "path"
                ]
            )
        ] = record


for record in size_candidates:

    plausible_paths[
        str(
            record[
                "path"
            ]
        )
    ] = record


hash_results = []


for key in sorted(
    plausible_paths
):

    record = plausible_paths[
        key
    ]

    path = record[
        "path"
    ]


    print(
        f"\nHashing:\n  {path}"
    )

    print(
        f"  size: {record['size_bytes']:,} B"
    )


    digest = sha256_file(
        path,
        progress=True,
    )


    matched_expected = [
        name
        for name, spec in EXPECTED.items()
        if digest
        ==
        spec[
            "sha256"
        ]
    ]


    print(
        "  SHA256:",
        digest
    )

    print(
        "  exact frozen identity:",
        (
            matched_expected
            if matched_expected
            else
            "NONE"
        ),
    )


    hash_results.append(
        {
            "path":
                str(
                    path
                ),

            "size_bytes":
                record[
                    "size_bytes"
                ],

            "sha256":
                digest,

            "matched_expected":
                matched_expected,
        }
    )


# =============================================================================
# 10. RESOLUTION SUMMARY
# =============================================================================

banner(
    "STAGE26-8C0-DIAG :: RESOLUTION SUMMARY"
)


resolved = {
    name: []
    for name in EXPECTED
}


for result in hash_results:

    for name in result[
        "matched_expected"
    ]:

        resolved[
            name
        ].append(
            result[
                "path"
            ]
        )


for name in EXPECTED:

    print(
        f"\n{name}"
    )

    print(
        "  exact SHA-identical candidate count:",
        len(
            resolved[
                name
            ]
        )
    )


    for path in resolved[
        name
    ]:

        print(
            "   ",
            path
        )


release_resolved = (
    len(
        resolved[
            "stage20-Monday-compact-corpus-v1.tar"
        ]
    )
    ==
    1
)


members_resolved = all(
    len(
        resolved[
            filename
        ]
    )
    ==
    1
    for filename in [
        "encoded_bytes.bin",
        "flow_offsets.npy",
        "packet_lengths.npy",
    ]
)


print(
    "\nExact release TAR resolved:",
    release_resolved
)

print(
    "All three exact members resolved directly:",
    members_resolved
)


if release_resolved:

    recommended_next = (
        "USE_SHA_IDENTICAL_RELEASE_TAR_FOR_STAGE26_8C0_RESTORE"
    )


elif members_resolved:

    recommended_next = (
        "USE_THREE_SHA_IDENTICAL_DIRECT_MEMBERS_FOR_LABEL_FREE_RESTORE"
    )


else:

    recommended_next = (
        "FROZEN_COMPACT_SOURCE_NOT_ATTACHED_TO_CURRENT_KAGGLE_RUNTIME"
    )


print(
    "\nDIAGNOSIS:"
)

print(
    " ",
    recommended_next
)


# =============================================================================
# 11. FINAL READ-ONLY CLOSURE
# =============================================================================

banner(
    "STAGE26-8C0-DIAG COMPLETE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed during diagnostic."
    )


if final_remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed during diagnostic."
    )


if final_status:

    raise RuntimeError(
        "Diagnostic unexpectedly modified Git."
    )


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  extraction performed      : NO"
)

print(
    "  compact files copied      : NO"
)

print(
    "  representation executed   : NO"
)

print(
    "  sensitivity timing        : NO"
)

print(
    "  sensitivity results seen  : NO"
)

print(
    "  historical 4C3 changed    : NO"
)

print(
    "  Stage26-7C changed        : NO"
)

print(
    "  model inference           : NO"
)

print(
    "  bootstrap                 : NO"
)

print(
    "  Pareto                    : NO"
)

print(
    "  PCAP                      : NO"
)

print(
    "  GPU                       : NO"
)

print(
    "  Git modified              : NO"
)


STAGE26-8C0-DIAG :: DURABLE STATE
Expected parent: 973e0c479ce91ad21e92ed5db11585f26a4c49ec
Local HEAD     : 973e0c479ce91ad21e92ed5db11585f26a4c49ec
origin/main    : 973e0c479ce91ad21e92ed5db11585f26a4c49ec
Repo clean     : True

STAGE26-8C0-DIAG :: INPUT ROOT INVENTORY
/kaggle/input exists: True
/kaggle/input/datasets exists: True

Immediate /kaggle/input entries:
  /kaggle/input/datasets DIR

STAGE26-8C0-DIAG :: RECURSIVE FILE INVENTORY
Total mounted files: 29

STAGE26-8C0-DIAG :: EXACT BASENAME SEARCH

stage20-Monday-compact-corpus-v1.tar: 0 match(es)

encoded_bytes.bin: 0 match(es)

flow_offsets.npy: 0 match(es)

packet_lengths.npy: 0 match(es)

labels.npy: 0 match(es)

STAGE26-8C0-DIAG :: EXACT FROZEN SIZE SEARCH
Files matching any frozen byte size: 0

STAGE26-8C0-DIAG :: ARCHIVE-LIKE FILES
Archive-like files: 0

STAGE26-8C0-DIAG :: SEMANTIC FILENAME SEARCH
Semantic filename/path candidates: 0

STAGE26-8C0-DIAG :: SHA256 OF PLAUSIBLE CANDIDATES

STAGE26-8C0-DIAG :: RESOLUTION SU

In [8]:
# =============================================================================
# STAGE26-8C0-DIAG2
# READ-ONLY REPOSITORY PROVENANCE SEARCH FOR THE FROZEN COMPACT RELEASE
#
# PURPOSE
# -------
# The current Kaggle runtime does not have the Stage20 compact corpus attached.
#
# Before downloading or restoring anything, locate the exact committed
# provenance for:
#
#   stage20-Monday-compact-corpus-v1.tar
#
# Expected:
#   size   = 595,261,440 bytes
#   SHA256 = 4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20
#
# SEARCHES
# --------
# - current tracked repository text
# - Git history patches
# - Git tags / release-like refs
# - committed URLs containing the release filename / SHA / tag clues
#
# THIS CELL DOES NOT:
# - download anything
# - restore anything
# - access PCAP
# - recreate a corpus
# - run V2/V3
# - perform timing
# - use GPU
# - modify Git
# =============================================================================

from __future__ import annotations

import re
import subprocess
from pathlib import Path


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "973e0c479ce91ad21e92ed5db11585f26a4c49ec"
)

RELEASE_FILENAME = (
    "stage20-Monday-compact-corpus-v1.tar"
)

RELEASE_SHA256 = (
    "4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20"
)

MEMBER_SHA256S = [
    "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",
    "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",
    "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
]


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            + " ".join(
                map(str, cmd)
            )
            + "\n\n"
            + p.stdout
        )

    return p.stdout


def git(*args, check=True):

    return run(
        [
            "git",
            *args,
        ],
        check=check,
    ).strip()


# =============================================================================
# 2. DURABLE STATE
# =============================================================================

banner(
    "STAGE26-8C0-DIAG2 :: DURABLE STATE"
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected local HEAD."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected origin/main."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


# =============================================================================
# 3. CURRENT TRACKED FILE SEARCH
# =============================================================================

banner(
    "STAGE26-8C0-DIAG2 :: CURRENT TRACKED REPOSITORY SEARCH"
)


tracked = git(
    "ls-files"
).splitlines()


search_terms = [
    RELEASE_FILENAME,
    RELEASE_SHA256,
    *MEMBER_SHA256S,
    "Monday-compact-corpus",
    "compact-corpus-v1",
    "stage20-Monday",
]


current_hits = []


for relpath in tracked:

    path = (
        REPO
        /
        relpath
    )


    if not path.is_file():

        continue


    # Avoid opening obvious large/binary artifacts.
    if path.stat().st_size > 20 * 1024 * 1024:

        continue


    try:

        text = path.read_text(
            encoding="utf-8",
            errors="ignore",
        )

    except Exception:

        continue


    for line_number, line in enumerate(
        text.splitlines(),
        start=1,
    ):

        matched = [
            term
            for term in search_terms
            if term.lower()
            in
            line.lower()
        ]


        if matched:

            current_hits.append(
                {
                    "path":
                        relpath,

                    "line":
                        line_number,

                    "matched":
                        matched,

                    "text":
                        line.strip(),
                }
            )


print(
    "Current tracked hits:",
    len(
        current_hits
    )
)


for hit in current_hits[
    :300
]:

    print(
        f"\n{hit['path']}:{hit['line']}"
    )

    print(
        "  matched:",
        hit[
            "matched"
        ]
    )

    print(
        "  text   :",
        hit[
            "text"
        ]
    )


# =============================================================================
# 4. EXTRACT URLS FROM RELEVANT CURRENT HITS
# =============================================================================

banner(
    "STAGE26-8C0-DIAG2 :: CURRENT URL CANDIDATES"
)


url_regex = re.compile(
    r"https?://[^\s\"'<>]+"
)


url_candidates = []


for hit in current_hits:

    for url in url_regex.findall(
        hit[
            "text"
        ]
    ):

        url_candidates.append(
            {
                "path":
                    hit[
                        "path"
                    ],

                "line":
                    hit[
                        "line"
                    ],

                "url":
                    url.rstrip(
                        ".,);]"
                    ),
            }
        )


# Also inspect surrounding relevant files for GitHub/HF release URLs.
relevant_files = sorted(
    set(
        hit[
            "path"
        ]
        for hit in current_hits
    )
)


for relpath in relevant_files:

    path = (
        REPO
        /
        relpath
    )


    text = path.read_text(
        encoding="utf-8",
        errors="ignore",
    )


    for line_number, line in enumerate(
        text.splitlines(),
        start=1,
    ):

        lower = line.lower()


        if (
            "github.com"
            in
            lower
            or
            "huggingface.co"
            in
            lower
            or
            "/releases/"
            in
            lower
            or
            "release asset"
            in
            lower
        ):

            urls = url_regex.findall(
                line
            )


            for url in urls:

                url_candidates.append(
                    {
                        "path":
                            relpath,

                        "line":
                            line_number,

                        "url":
                            url.rstrip(
                                ".,);]"
                            ),
                    }
                )


# Deduplicate.
seen_urls = set()
dedup_urls = []


for item in url_candidates:

    key = item[
        "url"
    ]


    if key in seen_urls:
        continue


    seen_urls.add(
        key
    )

    dedup_urls.append(
        item
    )


print(
    "Unique URL candidates:",
    len(
        dedup_urls
    )
)


for item in dedup_urls[
    :200
]:

    print(
        f"\n{item['path']}:{item['line']}"
    )

    print(
        " ",
        item[
            "url"
        ]
    )


# =============================================================================
# 5. GIT HISTORY — EXACT RELEASE FILENAME
# =============================================================================

banner(
    "STAGE26-8C0-DIAG2 :: GIT HISTORY — RELEASE FILENAME"
)


history_filename = git(
    "log",
    "--all",
    "--decorate",
    "--oneline",
    "-S",
    RELEASE_FILENAME,
    "--",
    ".",
    check=False,
)


print(
    history_filename
    if history_filename
    else
    "<no -S filename hits>"
)


# =============================================================================
# 6. GIT HISTORY — RELEASE SHA
# =============================================================================

banner(
    "STAGE26-8C0-DIAG2 :: GIT HISTORY — RELEASE SHA256"
)


history_sha = git(
    "log",
    "--all",
    "--decorate",
    "--oneline",
    "-S",
    RELEASE_SHA256,
    "--",
    ".",
    check=False,
)


print(
    history_sha
    if history_sha
    else
    "<no -S SHA hits>"
)


# =============================================================================
# 7. SHOW PATCH CONTEXT FOR RELEVANT COMMITS
# =============================================================================

banner(
    "STAGE26-8C0-DIAG2 :: RELEVANT COMMIT PATCH CONTEXT"
)


commit_candidates = []


for block in [
    history_filename,
    history_sha,
]:

    for line in block.splitlines():

        if not line.strip():
            continue


        commit = line.split(
            maxsplit=1
        )[
            0
        ]


        if re.fullmatch(
            r"[0-9a-fA-F]{7,40}",
            commit,
        ):

            if commit not in commit_candidates:

                commit_candidates.append(
                    commit
                )


print(
    "Relevant commit count:",
    len(
        commit_candidates
    )
)


for commit in commit_candidates[
    :30
]:

    print(
        "\n"
        +
        "-" * 124
    )

    print(
        "COMMIT:",
        commit
    )

    print(
        "-" * 124
    )


    patch = git(
        "show",
        "--format=fuller",
        "--stat",
        "--patch",
        commit,
        check=False,
    )


    # Print only lines relevant to release/source provenance,
    # with modest surrounding context.
    patch_lines = patch.splitlines()


    relevant_indices = []


    for i, line in enumerate(
        patch_lines
    ):

        lower = line.lower()


        if (
            RELEASE_FILENAME.lower()
            in
            lower
            or
            RELEASE_SHA256.lower()
            in
            lower
            or
            "github.com"
            in
            lower
            or
            "huggingface.co"
            in
            lower
            or
            "release"
            in
            lower
            or
            "compact corpus"
            in
            lower
        ):

            relevant_indices.append(
                i
            )


    printed = set()


    for index in relevant_indices:

        start = max(
            0,
            index - 4,
        )

        end = min(
            len(
                patch_lines
            ),
            index + 6,
        )


        for j in range(
            start,
            end,
        ):

            if j in printed:
                continue


            print(
                patch_lines[
                    j
                ]
            )

            printed.add(
                j
            )


# =============================================================================
# 8. TAG / REF INVENTORY
# =============================================================================

banner(
    "STAGE26-8C0-DIAG2 :: TAG / RELEASE-LIKE REF INVENTORY"
)


tags = git(
    "tag",
    "--list",
)


print(
    "Tags:"
)


if tags:

    for tag in tags.splitlines():

        print(
            " ",
            tag
        )

else:

    print(
        "  <none>"
    )


refs = git(
    "for-each-ref",
    "--format=%(refname:short)",
    "refs/remotes",
    "refs/tags",
)


print(
    "\nRefs containing stage20/release/compact:"
)


matching_refs = [
    ref
    for ref in refs.splitlines()
    if any(
        token
        in
        ref.lower()
        for token in [
            "stage20",
            "release",
            "compact",
            "monday",
        ]
    )
]


if matching_refs:

    for ref in matching_refs:

        print(
            " ",
            ref
        )

else:

    print(
        "  <none>"
    )


# =============================================================================
# 9. REMOTE CONFIG
# =============================================================================

banner(
    "STAGE26-8C0-DIAG2 :: GIT REMOTE"
)


print(
    git(
        "remote",
        "-v",
    )
)


# =============================================================================
# 10. RESOLUTION SUMMARY
# =============================================================================

banner(
    "STAGE26-8C0-DIAG2 :: RESOLUTION SUMMARY"
)


print(
    "Exact release filename current hits:",
    sum(
        RELEASE_FILENAME.lower()
        in
        hit[
            "text"
        ].lower()
        for hit in current_hits
    )
)

print(
    "Exact release SHA current hits     :",
    sum(
        RELEASE_SHA256.lower()
        in
        hit[
            "text"
        ].lower()
        for hit in current_hits
    )
)

print(
    "URL candidates                    :",
    len(
        dedup_urls
    )
)

print(
    "History commits                   :",
    len(
        commit_candidates
    )
)

print(
    "Release-like refs                 :",
    len(
        matching_refs
    )
)


print(
    "\nIMPORTANT:"
)

print(
    "  No download was attempted."
)

print(
    "  No corpus was recreated."
)

print(
    "  No PCAP was accessed."
)

print(
    "  No sensitivity timing was executed."
)


# =============================================================================
# 11. FINAL GIT CLOSURE
# =============================================================================

banner(
    "STAGE26-8C0-DIAG2 COMPLETE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed during provenance search."
    )


if final_remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed during provenance search."
    )


if final_status:

    raise RuntimeError(
        "Repository changed during provenance search."
    )


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  download attempted      : NO"
)

print(
    "  corpus recreated        : NO"
)

print(
    "  PCAP accessed           : NO"
)

print(
    "  representation executed : NO"
)

print(
    "  sensitivity timing      : NO"
)

print(
    "  results seen            : NO"
)

print(
    "  Stage26-7C changed      : NO"
)

print(
    "  Pareto                  : NO"
)

print(
    "  GPU                     : NO"
)

print(
    "  Git modified            : NO"
)


STAGE26-8C0-DIAG2 :: DURABLE STATE
Expected parent: 973e0c479ce91ad21e92ed5db11585f26a4c49ec
Local HEAD     : 973e0c479ce91ad21e92ed5db11585f26a4c49ec
origin/main    : 973e0c479ce91ad21e92ed5db11585f26a4c49ec
Repo clean     : True

STAGE26-8C0-DIAG2 :: CURRENT TRACKED REPOSITORY SEARCH
Current tracked hits: 123

docs/STAGE20_1E1M_MONDAY_SUPERVISED_COMPACT_CORPUS.md:176
  matched: ['27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c']
  text   : - `encoded_bytes.bin` — 522845159 bytes — SHA256 `27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c`

docs/STAGE20_1E1M_MONDAY_SUPERVISED_COMPACT_CORPUS.md:177
  matched: ['3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e']
  text   : - `flow_offsets.npy` — 4228208 bytes — SHA256 `3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e`

docs/STAGE20_1E1M_MONDAY_SUPERVISED_COMPACT_CORPUS.md:179
  matched: ['16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8']
  text   : - `pack

In [9]:
# =============================================================================
# STAGE26-8C0-R1
# DOWNLOAD EXACT EXISTING GITHUB RELEASE + RESTORE THREE LABEL-FREE MEMBERS
#
# DURABLE SCIENTIFIC PARENT:
#   973e0c479ce91ad21e92ed5db11585f26a4c49ec
#
# PROVENANCE NOW RESOLVED FROM COMMITTED EVIDENCE
# ------------------------------------------------
# GitHub Release:
#   tag:
#       stage20-compact-corpora-v1
#
#   asset:
#       stage20-Monday-compact-corpus-v1.tar
#
#   asset_id:
#       515837749
#
#   release_id:
#       371092195
#
#   bytes:
#       595,261,440
#
#   SHA256:
#       4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20
#
# The exact browser_download_url is taken from the COMMITTED Stage21
# release-upload receipt rather than invented here.
#
#
# RESTORE ONLY:
#   Monday/encoded_bytes.bin
#   Monday/flow_offsets.npy
#   Monday/packet_lengths.npy
#
# DO NOT EXTRACT:
#   Monday/labels.npy
#
#
# FINAL RUNTIME DIRECTORY:
#
#   /kaggle/working/stage26_deployment_profiling/
#       representation/stage26_4c2_equivalence/
#       Monday_release_representation_subset
#
#
# THIS CELL:
#   - verifies Git state
#   - verifies Stage26-8B locks
#   - verifies committed release provenance
#   - downloads the exact existing GitHub Release asset
#   - verifies full TAR byte size + SHA256 BEFORE extraction
#   - extracts exactly three label-free members
#   - verifies each restored byte size + SHA256
#   - verifies representation geometry
#   - writes a transient restoration receipt
#
# THIS CELL DOES NOT:
#   - recreate corpus from PCAP
#   - access PCAP
#   - extract labels.npy
#   - run V2
#   - run V3
#   - perform representation timing
#   - calculate sensitivity results
#   - run inference
#   - load models
#   - bootstrap
#   - recompute Stage26-7
#   - recompute Pareto
#   - use GPU
#   - modify Git
# =============================================================================

from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import requests


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "973e0c479ce91ad21e92ed5db11585f26a4c49ec"
)


# -----------------------------------------------------------------------------
# Exact committed release provenance
# -----------------------------------------------------------------------------

RELEASE_RECEIPT = (
    REPO
    / "results"
    / "stage21_architecture"
    / "stage21_1r1m_monday_release_upload_receipt.json"
)

EXPECTED_RELEASE_TAG = (
    "stage20-compact-corpora-v1"
)

EXPECTED_RELEASE_ID = (
    371092195
)

EXPECTED_ASSET_ID = (
    515837749
)

EXPECTED_RELEASE_FILENAME = (
    "stage20-Monday-compact-corpus-v1.tar"
)

EXPECTED_RELEASE_URL = (
    "https://github.com/themubasshir/"
    "ids2018-validation-safe-ablation/"
    "releases/download/"
    "stage20-compact-corpora-v1/"
    "stage20-Monday-compact-corpus-v1.tar"
)

EXPECTED_RELEASE_SIZE = (
    595_261_440
)

EXPECTED_RELEASE_SHA256 = (
    "4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20"
)


# -----------------------------------------------------------------------------
# Exact payload identities
# -----------------------------------------------------------------------------

EXPECTED_MEMBERS = {
    "encoded_bytes.bin": {
        "tar_member":
            "Monday/encoded_bytes.bin",

        "size_bytes":
            522_845_159,

        "sha256":
            "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",
    },

    "flow_offsets.npy": {
        "tar_member":
            "Monday/flow_offsets.npy",

        "size_bytes":
            4_228_208,

        "sha256":
            "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",
    },

    "packet_lengths.npy": {
        "tar_member":
            "Monday/packet_lengths.npy",

        "size_bytes":
            67_649_280,

        "sha256":
            "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
    },
}

FORBIDDEN_MEMBER = (
    "Monday/labels.npy"
)

FORBIDDEN_RUNTIME_FILE = (
    "labels.npy"
)

EXPECTED_FLOW_COUNT = (
    528_509
)


# -----------------------------------------------------------------------------
# Stage26-8B locks
# -----------------------------------------------------------------------------

LOCK_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_8b_representation_sensitivity_protocol_lock"
)

PROTOCOL = (
    LOCK_DIR
    / "stage26_8b_representation_sensitivity_protocol.json"
)

EXECUTION_PLAN = (
    LOCK_DIR
    / "stage26_8b_representation_sensitivity_execution_plan.json"
)

V3_WORKER = (
    LOCK_DIR
    / "stage26_representation_worker_v3_sensitivity.py"
)

FREEZE_RECEIPT = (
    LOCK_DIR
    / "stage26_8b_representation_sensitivity_freeze_receipt.json"
)

LOCK_MANIFEST = (
    LOCK_DIR
    / "stage26_8b_representation_sensitivity_lock_manifest.json"
)

V2_WORKER = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c1a_label_free_equivalence_erratum"
    / "stage26_representation_worker_v2.py"
)


EXPECTED_PROTOCOL_SHA256 = (
    "4ce96c1540b29bd9683305ed981412b46be4b0b2596dcb58b3064f0724f05c5e"
)

EXPECTED_EXECUTION_PLAN_SHA256 = (
    "5581bb2576919930913f49a308470083e7b27e315bc5cd99ba1fc1a853dcac14"
)

EXPECTED_V2_SHA256 = (
    "ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23"
)

EXPECTED_V3_SHA256 = (
    "c065e70fc16a628e482961f0db669d95affb841edd39153abf28dc99e868ad02"
)

EXPECTED_FREEZE_RECEIPT_SHA256 = (
    "d052bcd8bf3b2ed6a93f1ca3813036c030fa1d1de1fb28ffea166cf3e226bc33"
)

EXPECTED_LOCK_MANIFEST_SHA256 = (
    "6cfa4790ccabd1d755af40960b2e38846b4e15fe92ba2fac70d72ac3c162334f"
)


# -----------------------------------------------------------------------------
# Runtime paths
# -----------------------------------------------------------------------------

RUNTIME_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

SOURCE_CACHE = (
    RUNTIME_ROOT
    / "source_cache"
)

TAR_PATH = (
    SOURCE_CACHE
    / EXPECTED_RELEASE_FILENAME
)

PART_PATH = (
    SOURCE_CACHE
    / (
        EXPECTED_RELEASE_FILENAME
        +
        ".part"
    )
)

RESTORE_DIR = (
    RUNTIME_ROOT
    / "representation"
    / "stage26_4c2_equivalence"
    / "Monday_release_representation_subset"
)

RUNTIME_RECEIPT = (
    RUNTIME_ROOT
    / "cpu_closure"
    / "stage26_8c0_compact_input_restore_receipt.json"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            p.stdout
        )

    return p


def git(*args):

    return run(
        [
            "git",
            *args,
        ]
    ).stdout.strip()


def sha256_file(
    path,
    *,
    progress=False,
):

    path = Path(
        path
    )

    h = hashlib.sha256()

    total = 0

    next_report = (
        512 * 1024 * 1024
    )


    with path.open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

            total += len(
                block
            )


            if (
                progress
                and
                total >= next_report
            ):

                print(
                    f"  hashed {total / (1024**3):.2f} GiB"
                )

                next_report += (
                    512 * 1024 * 1024
                )


    return h.hexdigest()


def atomic_json(
    path,
    payload,
):

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(
            path
        )
        +
        ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )


    os.replace(
        tmp,
        path,
    )


# =============================================================================
# 2. DURABLE STATE
# =============================================================================

banner(
    "STAGE26-8C0-R1 :: DURABLE SCIENTIFIC STATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-8C0-R1 HEAD."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before restoration."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if RUNTIME_RECEIPT.exists():

    raise RuntimeError(
        "Stage26-8C0 restoration receipt already exists."
    )


# =============================================================================
# 3. EXACT STAGE26-8B LOCK IDENTITY
# =============================================================================

banner(
    "STAGE26-8C0-R1 :: EXACT STAGE26-8B LOCK IDENTITY"
)


lock_checks = [
    (
        "protocol",
        PROTOCOL,
        EXPECTED_PROTOCOL_SHA256,
    ),

    (
        "execution plan",
        EXECUTION_PLAN,
        EXPECTED_EXECUTION_PLAN_SHA256,
    ),

    (
        "V2 worker",
        V2_WORKER,
        EXPECTED_V2_SHA256,
    ),

    (
        "V3 worker",
        V3_WORKER,
        EXPECTED_V3_SHA256,
    ),

    (
        "freeze receipt",
        FREEZE_RECEIPT,
        EXPECTED_FREEZE_RECEIPT_SHA256,
    ),

    (
        "lock manifest",
        LOCK_MANIFEST,
        EXPECTED_LOCK_MANIFEST_SHA256,
    ),
]


for label, path, expected in lock_checks:

    actual = sha256_file(
        path
    )

    passed = (
        actual
        ==
        expected
    )


    print(
        f"{label:18s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Stage26-8B identity mismatch: {label}"
        )


# =============================================================================
# 4. COMMITTED RELEASE PROVENANCE GATE
# =============================================================================

banner(
    "STAGE26-8C0-R1 :: COMMITTED GITHUB RELEASE PROVENANCE"
)


release_receipt = json.loads(
    RELEASE_RECEIPT.read_text(
        encoding="utf-8"
    )
)


github_release = release_receipt[
    "github_release"
]

archive_asset = github_release[
    "archive_asset"
]


provenance_checks = {
    "status":
        (
            release_receipt[
                "status"
            ]
            ==
            "MONDAY_COMPACT_CORPUS_DURABLE_ON_GITHUB_RELEASE_AND_REMOTE_SHA_VERIFIED"
        ),

    "tag":
        (
            github_release[
                "tag"
            ]
            ==
            EXPECTED_RELEASE_TAG
        ),

    "release_id":
        (
            int(
                github_release[
                    "release_id"
                ]
            )
            ==
            EXPECTED_RELEASE_ID
        ),

    "asset_id":
        (
            int(
                archive_asset[
                    "asset_id"
                ]
            )
            ==
            EXPECTED_ASSET_ID
        ),

    "asset_name":
        (
            archive_asset[
                "name"
            ]
            ==
            EXPECTED_RELEASE_FILENAME
        ),

    "download_url":
        (
            archive_asset[
                "browser_download_url"
            ]
            ==
            EXPECTED_RELEASE_URL
        ),

    "bytes":
        (
            int(
                archive_asset[
                    "bytes"
                ]
            )
            ==
            EXPECTED_RELEASE_SIZE
        ),

    "remote_stream_bytes":
        (
            int(
                archive_asset[
                    "remote_stream_bytes"
                ]
            )
            ==
            EXPECTED_RELEASE_SIZE
        ),

    "local_sha256":
        (
            archive_asset[
                "local_sha256"
            ]
            ==
            EXPECTED_RELEASE_SHA256
        ),

    "remote_stream_sha256":
        (
            archive_asset[
                "remote_stream_sha256"
            ]
            ==
            EXPECTED_RELEASE_SHA256
        ),

    "byte_identity_verified":
        (
            archive_asset[
                "byte_identity_verified"
            ]
            is True
        ),
}


for name, passed in provenance_checks.items():

    print(
        f"{name:28s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    provenance_checks.values()
):

    raise RuntimeError(
        "Committed GitHub Release provenance mismatch."
    )


DOWNLOAD_URL = archive_asset[
    "browser_download_url"
]


print(
    "\nResolved committed release:"
)

print(
    "  tag      :",
    EXPECTED_RELEASE_TAG
)

print(
    "  release  :",
    EXPECTED_RELEASE_ID
)

print(
    "  asset id :",
    EXPECTED_ASSET_ID
)

print(
    "  filename :",
    EXPECTED_RELEASE_FILENAME
)

print(
    "  bytes    :",
    EXPECTED_RELEASE_SIZE
)

print(
    "  SHA256   :",
    EXPECTED_RELEASE_SHA256
)

print(
    "  URL      :",
    DOWNLOAD_URL
)


# =============================================================================
# 5. WORKSPACE CAPACITY
# =============================================================================

banner(
    "STAGE26-8C0-R1 :: WORKSPACE CAPACITY"
)


disk = shutil.disk_usage(
    "/kaggle/working"
)

free_gib = (
    disk.free
    /
    1024**3
)


print(
    f"Free /kaggle/working: {free_gib:.3f} GiB"
)


# TAR + restored files require roughly 1.1 GiB.
if disk.free < (
    2 * 1024**3
):

    raise RuntimeError(
        "Less than 2 GiB free; do not begin exact release restoration."
    )


# =============================================================================
# 6. DOWNLOAD EXACT RELEASE ASSET
# =============================================================================

banner(
    "STAGE26-8C0-R1 :: DOWNLOAD EXACT EXISTING RELEASE ASSET"
)


SOURCE_CACHE.mkdir(
    parents=True,
    exist_ok=True,
)


download_action = None


if TAR_PATH.exists():

    existing_size = int(
        TAR_PATH.stat().st_size
    )


    print(
        "Existing TAR detected."
    )

    print(
        "  size:",
        existing_size
    )


    if existing_size != EXPECTED_RELEASE_SIZE:

        raise RuntimeError(
            "Existing TAR has wrong size; refusing to overwrite automatically."
        )


    existing_sha = sha256_file(
        TAR_PATH,
        progress=True,
    )


    print(
        "  SHA256:",
        existing_sha
    )


    if existing_sha != EXPECTED_RELEASE_SHA256:

        raise RuntimeError(
            "Existing TAR has wrong SHA256; refusing to overwrite automatically."
        )


    download_action = (
        "REUSED_EXISTING_BYTE_IDENTICAL_RELEASE_TAR"
    )


else:

    # A stale .part file is only incomplete transport state, not scientific
    # evidence. Remove it before a fresh exact download.
    if PART_PATH.exists():

        print(
            "Removing stale incomplete transport file:"
        )

        print(
            " ",
            PART_PATH
        )

        PART_PATH.unlink()


    attempts = 3

    last_error = None


    for attempt in range(
        1,
        attempts + 1,
    ):

        print(
            f"\nDownload attempt {attempt}/{attempts}"
        )


        try:

            h = hashlib.sha256()

            total = 0

            next_report = (
                128 * 1024 * 1024
            )


            with requests.get(
                DOWNLOAD_URL,
                stream=True,
                allow_redirects=True,
                timeout=(
                    30,
                    120,
                ),
                headers={
                    "User-Agent":
                        "stage26-reproducibility-restore/1.0",
                },
            ) as response:

                print(
                    "  HTTP status:",
                    response.status_code
                )

                print(
                    "  final URL  :",
                    response.url
                )


                response.raise_for_status()


                content_length = response.headers.get(
                    "Content-Length"
                )


                if content_length is not None:

                    print(
                        "  Content-Length:",
                        content_length
                    )


                with PART_PATH.open(
                    "wb"
                ) as out:

                    for chunk in response.iter_content(
                        chunk_size=8 * 1024 * 1024
                    ):

                        if not chunk:
                            continue

                        out.write(
                            chunk
                        )

                        h.update(
                            chunk
                        )

                        total += len(
                            chunk
                        )


                        if total >= next_report:

                            print(
                                f"  downloaded "
                                f"{total / (1024**2):,.1f} MiB"
                            )

                            next_report += (
                                128 * 1024 * 1024
                            )


                    out.flush()

                    os.fsync(
                        out.fileno()
                    )


            digest = h.hexdigest()


            print(
                "  downloaded bytes:",
                total
            )

            print(
                "  downloaded SHA256:",
                digest
            )


            if total != EXPECTED_RELEASE_SIZE:

                raise RuntimeError(
                    f"Downloaded byte count mismatch: "
                    f"{total} != {EXPECTED_RELEASE_SIZE}"
                )


            if digest != EXPECTED_RELEASE_SHA256:

                raise RuntimeError(
                    "Downloaded release SHA256 mismatch."
                )


            os.replace(
                PART_PATH,
                TAR_PATH,
            )


            download_action = (
                "DOWNLOADED_FROM_COMMITTED_GITHUB_RELEASE_URL"
            )

            break


        except Exception as exc:

            last_error = exc


            print(
                "  attempt failed:",
                repr(
                    exc
                )
            )


            if PART_PATH.exists():

                PART_PATH.unlink()


            if attempt < attempts:

                print(
                    "  transport retry in 5 seconds..."
                )

                time.sleep(
                    5
                )


    if download_action is None:

        raise RuntimeError(
            "Exact GitHub release download failed after transport retries."
        ) from last_error


# =============================================================================
# 7. FULL RELEASE IDENTITY — BEFORE EXTRACTION
# =============================================================================

banner(
    "STAGE26-8C0-R1 :: FULL RELEASE BYTE IDENTITY"
)


tar_size = int(
    TAR_PATH.stat().st_size
)

tar_sha = sha256_file(
    TAR_PATH,
    progress=True,
)


print(
    "Expected bytes:",
    EXPECTED_RELEASE_SIZE
)

print(
    "Actual bytes  :",
    tar_size
)

print(
    "Expected SHA  :",
    EXPECTED_RELEASE_SHA256
)

print(
    "Actual SHA    :",
    tar_sha
)


if tar_size != EXPECTED_RELEASE_SIZE:

    raise RuntimeError(
        "Release TAR size mismatch."
    )


if tar_sha != EXPECTED_RELEASE_SHA256:

    raise RuntimeError(
        "Release TAR SHA256 mismatch."
    )


print(
    "\nFULL RELEASE IDENTITY: PASS"
)


# =============================================================================
# 8. TAR STRUCTURE — NO EXTRACTION YET
# =============================================================================

banner(
    "STAGE26-8C0-R1 :: TAR MEMBER IDENTITY"
)


with tarfile.open(
    TAR_PATH,
    mode="r:",
) as tf:

    members = {
        member.name:
            member
        for member in tf.getmembers()
        if member.isfile()
    }


for filename, spec in EXPECTED_MEMBERS.items():

    member_name = spec[
        "tar_member"
    ]


    if member_name not in members:

        raise RuntimeError(
            f"Required TAR member missing: {member_name}"
        )


    member = members[
        member_name
    ]


    print(
        f"{member_name:32s} "
        f"{member.size:12,d} B"
    )


    if int(
        member.size
    ) != int(
        spec[
            "size_bytes"
        ]
    ):

        raise RuntimeError(
            f"TAR member size mismatch: {member_name}"
        )


print(
    f"\n{FORBIDDEN_MEMBER} present in TAR:",
    FORBIDDEN_MEMBER in members
)

print(
    "labels extraction allowed:",
    False
)


# It is expected to exist in the authoritative archive,
# but it will not be extracted.
if FORBIDDEN_MEMBER not in members:

    raise RuntimeError(
        "Authoritative TAR unexpectedly lacks historical labels.npy member."
    )


# =============================================================================
# 9. CONSERVATIVE RESTORE TARGET HANDLING
# =============================================================================

banner(
    "STAGE26-8C0-R1 :: LABEL-FREE RESTORE"
)


print(
    "Restore directory:"
)

print(
    " ",
    RESTORE_DIR
)

print(
    "Already exists:",
    RESTORE_DIR.exists()
)


if RESTORE_DIR.exists():

    existing_files = sorted(
        path.name
        for path in RESTORE_DIR.iterdir()
        if path.is_file()
    )


    print(
        "Existing files:",
        existing_files
    )


    allowed_names = set(
        EXPECTED_MEMBERS
    )


    if set(
        existing_files
    ) != allowed_names:

        raise RuntimeError(
            "Existing restore directory is not exactly the required "
            "three-file label-free subset."
        )


    restored_action = (
        "REUSED_EXISTING_BYTE_IDENTICAL_LABEL_FREE_RESTORE"
    )


else:

    RESTORE_DIR.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    with tempfile.TemporaryDirectory(
        prefix="stage26_8c0_exact_restore_",
        dir=RESTORE_DIR.parent,
    ) as tmp_name:

        tmp_dir = Path(
            tmp_name
        )


        with tarfile.open(
            TAR_PATH,
            mode="r:",
        ) as tf:

            for filename, spec in EXPECTED_MEMBERS.items():

                member_name = spec[
                    "tar_member"
                ]

                member = tf.getmember(
                    member_name
                )


                source = tf.extractfile(
                    member
                )


                if source is None:

                    raise RuntimeError(
                        f"Could not open TAR member: {member_name}"
                    )


                destination = (
                    tmp_dir
                    /
                    filename
                )


                print(
                    f"\nExtracting {member_name}"
                )


                h = hashlib.sha256()

                written = 0


                with source, destination.open(
                    "wb"
                ) as out:

                    while True:

                        block = source.read(
                            8 * 1024 * 1024
                        )


                        if not block:
                            break


                        out.write(
                            block
                        )

                        h.update(
                            block
                        )

                        written += len(
                            block
                        )


                    out.flush()

                    os.fsync(
                        out.fileno()
                    )


                digest = h.hexdigest()


                print(
                    "  bytes :",
                    written
                )

                print(
                    "  SHA256:",
                    digest
                )


                if written != int(
                    spec[
                        "size_bytes"
                    ]
                ):

                    raise RuntimeError(
                        f"{filename}: restored size mismatch."
                    )


                if digest != spec[
                    "sha256"
                ]:

                    raise RuntimeError(
                        f"{filename}: restored SHA256 mismatch."
                    )


        if (
            tmp_dir
            /
            FORBIDDEN_RUNTIME_FILE
        ).exists():

            raise RuntimeError(
                "Forbidden labels.npy was restored."
            )


        os.replace(
            tmp_dir,
            RESTORE_DIR,
        )


    restored_action = (
        "EXTRACTED_EXACT_THREE_MEMBERS_FROM_VERIFIED_RELEASE_TAR"
    )


# =============================================================================
# 10. FINAL MEMBER HASH AUDIT
# =============================================================================

banner(
    "STAGE26-8C0-R1 :: FINAL RESTORED MEMBER AUDIT"
)


restored_identity = {}


for filename, spec in EXPECTED_MEMBERS.items():

    path = (
        RESTORE_DIR
        /
        filename
    )


    if not path.is_file():

        raise FileNotFoundError(
            path
        )


    size = int(
        path.stat().st_size
    )

    digest = sha256_file(
        path
    )


    passed = (
        size
        ==
        int(
            spec[
                "size_bytes"
            ]
        )
        and
        digest
        ==
        spec[
            "sha256"
        ]
    )


    print(
        f"{filename:24s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{size:12,d} B "
        f"{digest}"
    )


    if not passed:

        raise RuntimeError(
            f"Final member identity mismatch: {filename}"
        )


    restored_identity[
        filename
    ] = {
        "source_tar_member":
            spec[
                "tar_member"
            ],

        "size_bytes":
            size,

        "sha256":
            digest,
    }


labels_path = (
    RESTORE_DIR
    /
    FORBIDDEN_RUNTIME_FILE
)


print(
    "\nlabels.npy present in runtime:",
    labels_path.exists()
)


if labels_path.exists():

    raise RuntimeError(
        "labels.npy exists in label-free sensitivity runtime."
    )


runtime_files = sorted(
    path.name
    for path in RESTORE_DIR.iterdir()
    if path.is_file()
)


if runtime_files != sorted(
    EXPECTED_MEMBERS.keys()
):

    raise RuntimeError(
        "Runtime directory contains unexpected files."
    )


print(
    "Runtime file count:",
    len(
        runtime_files
    )
)

print(
    "Runtime members:",
    runtime_files
)


# =============================================================================
# 11. STRUCTURAL GEOMETRY AUDIT
# =============================================================================

banner(
    "STAGE26-8C0-R1 :: STRUCTURAL GEOMETRY"
)


encoded_path = (
    RESTORE_DIR
    /
    "encoded_bytes.bin"
)

offsets_path = (
    RESTORE_DIR
    /
    "flow_offsets.npy"
)

lengths_path = (
    RESTORE_DIR
    /
    "packet_lengths.npy"
)


offsets = np.load(
    offsets_path,
    mmap_mode="r",
    allow_pickle=False,
)

lengths = np.load(
    lengths_path,
    mmap_mode="r",
    allow_pickle=False,
)


print(
    "encoded bytes       :",
    encoded_path.stat().st_size
)

print(
    "flow_offsets dtype  :",
    offsets.dtype
)

print(
    "flow_offsets shape  :",
    offsets.shape
)

print(
    "packet_lengths dtype:",
    lengths.dtype
)

print(
    "packet_lengths shape:",
    lengths.shape
)


if offsets.ndim != 1:

    raise RuntimeError(
        "flow_offsets must be one-dimensional."
    )


if int(
    offsets.shape[
        0
    ]
) != (
    EXPECTED_FLOW_COUNT
    +
    1
):

    raise RuntimeError(
        "flow_offsets length != frozen flow_count + 1."
    )


if int(
    offsets[
        0
    ]
) != 0:

    raise RuntimeError(
        "flow_offsets[0] != 0."
    )


if int(
    offsets[
        -1
    ]
) != int(
    encoded_path.stat().st_size
):

    raise RuntimeError(
        "Final flow offset != encoded byte count."
    )


if lengths.ndim != 2:

    raise RuntimeError(
        "packet_lengths must be two-dimensional."
    )


if int(
    lengths.shape[
        0
    ]
) != EXPECTED_FLOW_COUNT:

    raise RuntimeError(
        "packet_lengths first dimension != frozen flow count."
    )


if int(
    lengths.shape[
        1
    ]
) != 64:

    raise RuntimeError(
        "packet_lengths second dimension != 64."
    )


print(
    "\nFrozen flow count:",
    EXPECTED_FLOW_COUNT
)

print(
    "Structural geometry: PASS"
)


# Release mmap handles before future fresh worker processes.
del offsets
del lengths


# =============================================================================
# 12. VERIFY HISTORICAL 4C3 CONFIG PATH COMPATIBILITY
# =============================================================================

banner(
    "STAGE26-8C0-R1 :: HISTORICAL 4C3 PATH COMPATIBILITY"
)


C4_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c3_cpu1_representation_timing"
)


for batch in [
    1,
    64,
    256,
    1024,
    8192,
]:

    configs = sorted(
        C4_DIR.glob(
            f"conditions/batch_{batch}/*_config.json"
        )
    )


    if len(
        configs
    ) != 1:

        raise RuntimeError(
            f"B={batch}: historical config resolution failure."
        )


    cfg = json.loads(
        configs[
            0
        ].read_text(
            encoding="utf-8"
        )
    )


    frozen_path = Path(
        cfg[
            "corpus_dir"
        ]
    )


    passed = (
        frozen_path
        ==
        RESTORE_DIR
    )


    print(
        f"B={batch:5d} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{frozen_path}"
    )


    if not passed:

        raise RuntimeError(
            f"B={batch}: historical corpus_dir mismatch."
        )


# =============================================================================
# 13. TRANSIENT RESTORATION RECEIPT
# =============================================================================

banner(
    "STAGE26-8C0-R1 :: TRANSIENT RESTORATION RECEIPT"
)


receipt = {
    "schema":
        "stage26_8c0_compact_input_restore_receipt_v2",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "status":
        "PASS_EXACT_GITHUB_RELEASE_LABEL_FREE_RESTORE",

    "release_provenance": {
        "source":
            "EXISTING_GITHUB_RELEASE",

        "committed_receipt":
            str(
                RELEASE_RECEIPT.relative_to(
                    REPO
                )
            ),

        "tag":
            EXPECTED_RELEASE_TAG,

        "release_id":
            EXPECTED_RELEASE_ID,

        "asset_id":
            EXPECTED_ASSET_ID,

        "asset":
            EXPECTED_RELEASE_FILENAME,

        "browser_download_url":
            DOWNLOAD_URL,

        "size_bytes":
            tar_size,

        "sha256":
            tar_sha,

        "download_action":
            download_action,

        "recreated_from_pcap":
            False,
    },

    "runtime": {
        "release_tar":
            str(
                TAR_PATH
            ),

        "restore_directory":
            str(
                RESTORE_DIR
            ),

        "restore_action":
            restored_action,

        "files":
            restored_identity,

        "file_count":
            3,

        "labels_file_present":
            False,
    },

    "geometry": {
        "flow_count":
            EXPECTED_FLOW_COUNT,

        "encoded_bytes":
            EXPECTED_MEMBERS[
                "encoded_bytes.bin"
            ][
                "size_bytes"
            ],

        "rows":
            64,

        "cols":
            256,

        "packet_slots":
            64,
    },

    "stage26_8b_identity": {
        "protocol_sha256":
            EXPECTED_PROTOCOL_SHA256,

        "execution_plan_sha256":
            EXPECTED_EXECUTION_PLAN_SHA256,

        "V2_worker_sha256":
            EXPECTED_V2_SHA256,

        "V3_worker_sha256":
            EXPECTED_V3_SHA256,

        "freeze_receipt_sha256":
            EXPECTED_FREEZE_RECEIPT_SHA256,

        "lock_manifest_sha256":
            EXPECTED_LOCK_MANIFEST_SHA256,
    },

    "scientific_state": {
        "release_downloaded_or_reused":
            True,

        "release_byte_identity_verified":
            True,

        "release_corpus_regenerated":
            False,

        "PCAP_accessed":
            False,

        "labels_extracted":
            False,

        "representation_executed":
            False,

        "sensitivity_timing_executed":
            False,

        "sensitivity_results_seen":
            False,

        "historical_4C3_modified":
            False,

        "stage26_7c_modified":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "bootstrap_executed":
            False,

        "Pareto_recomputed":
            False,

        "GPU_used":
            False,

        "Git_modified":
            False,
    },

    "next":
        (
            "Execute only the remotely anchored Stage26-8B randomized "
            "25-pair / 50-fresh-process contemporaneous V2-vs-V3 "
            "CPU1 representation sensitivity plan."
        ),
}


atomic_json(
    RUNTIME_RECEIPT,
    receipt,
)


receipt_sha = sha256_file(
    RUNTIME_RECEIPT
)


print(
    "Receipt:"
)

print(
    " ",
    RUNTIME_RECEIPT
)

print(
    "SHA256:"
)

print(
    " ",
    receipt_sha
)


# =============================================================================
# 14. FINAL GIT / SCIENTIFIC CLOSURE
# =============================================================================

banner(
    "STAGE26-8C0-R1 EXACT RELEASE RESTORE COMPLETE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed during release restoration."
    )


if final_remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed during release restoration."
    )


if final_status:

    raise RuntimeError(
        "Release restoration unexpectedly modified Git."
    )


print(
    "\nRELEASE PROVENANCE:"
)

print(
    "  existing GitHub Release : VERIFIED"
)

print(
    "  tag                     :",
    EXPECTED_RELEASE_TAG
)

print(
    "  asset ID                :",
    EXPECTED_ASSET_ID
)

print(
    "  full TAR bytes          : VERIFIED"
)

print(
    "  full TAR SHA256         : VERIFIED"
)

print(
    "  recreated from PCAP     : NO"
)


print(
    "\nLABEL-FREE RESTORE:"
)

print(
    "  encoded_bytes.bin  : VERIFIED"
)

print(
    "  flow_offsets.npy   : VERIFIED"
)

print(
    "  packet_lengths.npy : VERIFIED"
)

print(
    "  labels.npy         : NOT EXTRACTED"
)

print(
    "  exact runtime path : VERIFIED"
)

print(
    "  flow geometry      : VERIFIED"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  V2 executed            : NO"
)

print(
    "  V3 executed            : NO"
)

print(
    "  representation timing  : NO"
)

print(
    "  sensitivity results    : NOT SEEN"
)

print(
    "  historical 4C3 changed : NO"
)

print(
    "  Stage26-7C changed     : NO"
)

print(
    "  inference              : NO"
)

print(
    "  bootstrap              : NO"
)

print(
    "  Pareto                 : NO"
)

print(
    "  PCAP                   : NO"
)

print(
    "  GPU                    : NO"
)

print(
    "  Git modified           : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Exact inputs are now eligible for the already-frozen"
)

print(
    "  Stage26-8B paired V2/V3 sensitivity execution:"
)

print(
    "    25 randomized pair blocks"
)

print(
    "    50 fresh processes"
)

print(
    "    CPU1 only"
)

print(
    "    no replacement pairs"
)

print(
    "    no bootstrap"
)

print(
    "    GPU OFF"
)


STAGE26-8C0-R1 :: DURABLE SCIENTIFIC STATE
Expected parent: 973e0c479ce91ad21e92ed5db11585f26a4c49ec
Local HEAD     : 973e0c479ce91ad21e92ed5db11585f26a4c49ec
origin/main    : 973e0c479ce91ad21e92ed5db11585f26a4c49ec
Repo clean     : True

STAGE26-8C0-R1 :: EXACT STAGE26-8B LOCK IDENTITY
protocol           PASS 4ce96c1540b29bd9683305ed981412b46be4b0b2596dcb58b3064f0724f05c5e
execution plan     PASS 5581bb2576919930913f49a308470083e7b27e315bc5cd99ba1fc1a853dcac14
V2 worker          PASS ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23
V3 worker          PASS c065e70fc16a628e482961f0db669d95affb841edd39153abf28dc99e868ad02
freeze receipt     PASS d052bcd8bf3b2ed6a93f1ca3813036c030fa1d1de1fb28ffea166cf3e226bc33
lock manifest      PASS 6cfa4790ccabd1d755af40960b2e38846b4e15fe92ba2fac70d72ac3c162334f

STAGE26-8C0-R1 :: COMMITTED GITHUB RELEASE PROVENANCE
status                      : PASS
tag                         : PASS
release_id                  : PASS
asset_id        

In [10]:
# =============================================================================
# STAGE26-8C1
# EXECUTE FROZEN PAIRED V2-vs-V3 REPRESENTATION SENSITIVITY
#
# DURABLE SCIENTIFIC PARENT:
#   973e0c479ce91ad21e92ed5db11585f26a4c49ec
#
# FROZEN BY STAGE26-8B:
#
#   25 randomized pair blocks
#   5 pair replicates / batch
#   50 fresh worker processes
#   CPU1 only
#   batches = [1, 64, 256, 1024, 8192]
#   RNG = numpy.random.default_rng(26042), PCG64
#   no adaptive reordering
#   no replacement pairs
#
# A:
#   exact historical V2 worker
#
# B:
#   frozen clean V3 sensitivity worker
#
# PRIMARY ESTIMAND:
#
#   median throughput V3 / median throughput V2
#
# SECONDARY:
#
#   p50 V3 / p50 V2
#   p95 V3 / p95 V2
#   p99 V3 / p99 V2 when n >= 100
#
# NO:
#   - bootstrap
#   - hypothesis test
#   - significance threshold
#   - materiality threshold
#   - historical 4C3 replacement
#   - Stage26-7C recomputation
#   - model inference
#   - Pareto recomputation
#   - PCAP
#   - labels
#   - GPU
#   - Git modification
#
# IMPORTANT:
# Results are written TRANSIENTLY outside Git first.
# We inspect them before creating the durable results checkpoint.
# =============================================================================

from __future__ import annotations

import copy
import csv
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import psutil


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "973e0c479ce91ad21e92ed5db11585f26a4c49ec"
)


# -----------------------------------------------------------------------------
# Stage26-8B frozen lock
# -----------------------------------------------------------------------------

LOCK_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_8b_representation_sensitivity_protocol_lock"
)

PROTOCOL = (
    LOCK_DIR
    / "stage26_8b_representation_sensitivity_protocol.json"
)

EXECUTION_PLAN = (
    LOCK_DIR
    / "stage26_8b_representation_sensitivity_execution_plan.json"
)

FREEZE_RECEIPT = (
    LOCK_DIR
    / "stage26_8b_representation_sensitivity_freeze_receipt.json"
)

LOCK_MANIFEST = (
    LOCK_DIR
    / "stage26_8b_representation_sensitivity_lock_manifest.json"
)

V3_WORKER = (
    LOCK_DIR
    / "stage26_representation_worker_v3_sensitivity.py"
)


EXPECTED_PROTOCOL_SHA256 = (
    "4ce96c1540b29bd9683305ed981412b46be4b0b2596dcb58b3064f0724f05c5e"
)

EXPECTED_EXECUTION_PLAN_SHA256 = (
    "5581bb2576919930913f49a308470083e7b27e315bc5cd99ba1fc1a853dcac14"
)

EXPECTED_FREEZE_RECEIPT_SHA256 = (
    "d052bcd8bf3b2ed6a93f1ca3813036c030fa1d1de1fb28ffea166cf3e226bc33"
)

EXPECTED_LOCK_MANIFEST_SHA256 = (
    "6cfa4790ccabd1d755af40960b2e38846b4e15fe92ba2fac70d72ac3c162334f"
)

EXPECTED_V3_SHA256 = (
    "c065e70fc16a628e482961f0db669d95affb841edd39153abf28dc99e868ad02"
)


# -----------------------------------------------------------------------------
# Historical exact V2
# -----------------------------------------------------------------------------

V2_WORKER = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c1a_label_free_equivalence_erratum"
    / "stage26_representation_worker_v2.py"
)

EXPECTED_V2_SHA256 = (
    "ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23"
)


# -----------------------------------------------------------------------------
# Historical 4C3 configs
# -----------------------------------------------------------------------------

C4_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c3_cpu1_representation_timing"
)


# -----------------------------------------------------------------------------
# Exact restored runtime input
# -----------------------------------------------------------------------------

RUNTIME_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

CORPUS_DIR = (
    RUNTIME_ROOT
    / "representation"
    / "stage26_4c2_equivalence"
    / "Monday_release_representation_subset"
)

RESTORE_RECEIPT = (
    RUNTIME_ROOT
    / "cpu_closure"
    / "stage26_8c0_compact_input_restore_receipt.json"
)

EXPECTED_RESTORE_RECEIPT_SHA256 = (
    "babcf2331db7c242297b346bad0904bb061251acb83003b5222c16295de96ff1"
)


EXPECTED_RUNTIME_FILES = {
    "encoded_bytes.bin": {
        "size_bytes":
            522_845_159,
    },

    "flow_offsets.npy": {
        "size_bytes":
            4_228_208,
    },

    "packet_lengths.npy": {
        "size_bytes":
            67_649_280,
    },
}


# -----------------------------------------------------------------------------
# Execution policy
# -----------------------------------------------------------------------------

CPU_AFFINITY = [
    0,
]

THREAD_COUNT = 1

MAX_CPU_PERCENT = 20.0

MIN_AVAILABLE_RAM_GIB = 8.0

ENVIRONMENT_GATE_ATTEMPTS = 3

ENVIRONMENT_GATE_RETRY_SECONDS = 5

WORKER_TIMEOUT_SECONDS = 600


BATCHES = [
    1,
    64,
    256,
    1024,
    8192,
]


# -----------------------------------------------------------------------------
# Transient output
# -----------------------------------------------------------------------------

OUT = (
    RUNTIME_ROOT
    / "cpu_closure"
    / "stage26_8c1_paired_representation_sensitivity"
)

CONFIG_DIR = (
    OUT
    / "configs"
)

RESULT_DIR = (
    OUT
    / "worker_results"
)

LOG_DIR = (
    OUT
    / "worker_logs"
)

WORKER_EXECUTION_CSV = (
    OUT
    / "stage26_8c1_worker_executions.csv"
)

RAW_OBSERVATIONS_CSV = (
    OUT
    / "stage26_8c1_raw_observations.csv"
)

PAIR_RESULTS_CSV = (
    OUT
    / "stage26_8c1_pair_results.csv"
)

BATCH_SUMMARY_CSV = (
    OUT
    / "stage26_8c1_batch_sensitivity_summary.csv"
)

RESULTS_JSON = (
    OUT
    / "stage26_8c1_paired_sensitivity_results.json"
)

EXECUTION_RECEIPT = (
    OUT
    / "stage26_8c1_execution_receipt.json"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def git(*args):

    p = subprocess.run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def atomic_json(path, payload):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        +
        ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )


    os.replace(
        tmp,
        path,
    )


def write_csv(
    path,
    rows,
    fieldnames,
):

    with Path(path).open(
        "w",
        encoding="utf-8",
        newline="",
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        writer.writerows(
            rows
        )


def finite_positive(
    value,
    label,
):

    x = float(
        value
    )

    if (
        not math.isfinite(
            x
        )
        or
        x <= 0
    ):

        raise RuntimeError(
            f"{label}: expected finite positive value; got {value!r}"
        )

    return x


# =============================================================================
# 2. DURABLE GIT GATE
# =============================================================================

banner(
    "STAGE26-8C1 :: DURABLE SCIENTIFIC STATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-8C1 HEAD."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before sensitivity execution."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if OUT.exists():

    raise RuntimeError(
        "Stage26-8C1 transient output already exists."
    )


# =============================================================================
# 3. EXACT STAGE26-8B LOCK GATE
# =============================================================================

banner(
    "STAGE26-8C1 :: EXACT FROZEN SENSITIVITY LOCK"
)


lock_checks = [
    (
        "protocol",
        PROTOCOL,
        EXPECTED_PROTOCOL_SHA256,
    ),

    (
        "execution plan",
        EXECUTION_PLAN,
        EXPECTED_EXECUTION_PLAN_SHA256,
    ),

    (
        "freeze receipt",
        FREEZE_RECEIPT,
        EXPECTED_FREEZE_RECEIPT_SHA256,
    ),

    (
        "lock manifest",
        LOCK_MANIFEST,
        EXPECTED_LOCK_MANIFEST_SHA256,
    ),

    (
        "V2 worker",
        V2_WORKER,
        EXPECTED_V2_SHA256,
    ),

    (
        "V3 worker",
        V3_WORKER,
        EXPECTED_V3_SHA256,
    ),
]


for label, path, expected in lock_checks:

    actual = sha256_file(
        path
    )

    passed = (
        actual
        ==
        expected
    )


    print(
        f"{label:18s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Frozen sensitivity identity mismatch: {label}"
        )


protocol = json.loads(
    PROTOCOL.read_text(
        encoding="utf-8"
    )
)

plan = json.loads(
    EXECUTION_PLAN.read_text(
        encoding="utf-8"
    )
)


if protocol[
    "status"
] != "FROZEN_BEFORE_ANY_SENSITIVITY_TIMING":

    raise RuntimeError(
        "Sensitivity protocol was not prospectively frozen."
    )


if protocol[
    "experimental_design"
][
    "type"
] != "PAIRED_CONTEMPORANEOUS_A_B_SENSITIVITY":

    raise RuntimeError(
        "Unexpected sensitivity design."
    )


if protocol[
    "inference_policy"
][
    "bootstrap"
] is not False:

    raise RuntimeError(
        "Bootstrap unexpectedly permitted."
    )


if protocol[
    "inference_policy"
][
    "materiality_threshold"
] is not None:

    raise RuntimeError(
        "Materiality threshold unexpectedly defined."
    )


pair_blocks = plan[
    "pair_blocks"
]


if len(
    pair_blocks
) != 25:

    raise RuntimeError(
        "Expected exactly 25 frozen pair blocks."
    )


# Exact geometry of randomized plan.
block_ids = [
    int(
        block[
            "pair_block_execution_index"
        ]
    )
    for block in pair_blocks
]


if block_ids != list(
    range(
        1,
        26,
    )
):

    raise RuntimeError(
        "Frozen pair-block execution indices are not exactly 1..25."
    )


plan_counts = Counter(
    int(
        block[
            "batch_size"
        ]
    )
    for block in pair_blocks
)


if plan_counts != Counter(
    {
        1:
            5,

        64:
            5,

        256:
            5,

        1024:
            5,

        8192:
            5,
    }
):

    raise RuntimeError(
        f"Frozen batch/pair geometry mismatch: {plan_counts}"
    )


for block in pair_blocks:

    order = block[
        "within_pair_order"
    ]


    if sorted(
        order
    ) != [
        "V2_ORIGINAL",
        "V3_CLEAN",
    ]:

        raise RuntimeError(
            f"Invalid frozen within-pair order: {order}"
        )


print(
    "\nPair blocks      :",
    len(
        pair_blocks
    )
)

print(
    "Worker processes :",
    len(
        pair_blocks
    )
    *
    2
)

print(
    "Batches          :",
    BATCHES
)

print(
    "Randomization    : FROZEN"
)


# =============================================================================
# 4. RESTORED INPUT RECEIPT + NON-PERTURBING SOURCE GATE
# =============================================================================

banner(
    "STAGE26-8C1 :: RESTORED INPUT GATE"
)


if not RESTORE_RECEIPT.is_file():

    raise FileNotFoundError(
        RESTORE_RECEIPT
    )


restore_receipt_sha = sha256_file(
    RESTORE_RECEIPT
)


print(
    "Restore receipt SHA256:"
)

print(
    " ",
    restore_receipt_sha
)


if restore_receipt_sha != EXPECTED_RESTORE_RECEIPT_SHA256:

    raise RuntimeError(
        "Stage26-8C0 restoration receipt mismatch."
    )


restore_receipt = json.loads(
    RESTORE_RECEIPT.read_text(
        encoding="utf-8"
    )
)


if restore_receipt[
    "status"
] != "PASS_EXACT_GITHUB_RELEASE_LABEL_FREE_RESTORE":

    raise RuntimeError(
        "Exact compact restore did not PASS."
    )


if restore_receipt[
    "scientific_state"
][
    "labels_extracted"
] is not False:

    raise RuntimeError(
        "Labels were unexpectedly extracted."
    )


if restore_receipt[
    "scientific_state"
][
    "release_corpus_regenerated"
] is not False:

    raise RuntimeError(
        "Release corpus was unexpectedly regenerated."
    )


expected_names = sorted(
    EXPECTED_RUNTIME_FILES
)

actual_names = sorted(
    p.name
    for p in CORPUS_DIR.iterdir()
    if p.is_file()
)


print(
    "Expected runtime files:",
    expected_names
)

print(
    "Actual runtime files  :",
    actual_names
)


if actual_names != expected_names:

    raise RuntimeError(
        "Runtime representation source file set changed."
    )


for filename, spec in EXPECTED_RUNTIME_FILES.items():

    path = (
        CORPUS_DIR
        /
        filename
    )


    size = int(
        path.stat().st_size
    )


    print(
        f"{filename:24s} "
        f"{size:12,d} B "
        f"{'PASS' if size == spec['size_bytes'] else 'FAIL'}"
    )


    if size != int(
        spec[
            "size_bytes"
        ]
    ):

        raise RuntimeError(
            f"Runtime source size changed: {filename}"
        )


if (
    CORPUS_DIR
    /
    "labels.npy"
).exists():

    raise RuntimeError(
        "labels.npy exists in sensitivity runtime."
    )


print(
    "\nFull encoded_bytes.bin rehash immediately before timing: NO"
)

print(
    "Reason: exact byte identity already certified by Stage26-8C0;"
)

print(
    "        avoid additional pre-measurement page-cache perturbation."
)


# =============================================================================
# 5. CPU / THREAD EXECUTION PRE-FLIGHT
# =============================================================================

banner(
    "STAGE26-8C1 :: CPU1 EXECUTION PREFLIGHT"
)


if not hasattr(
    os,
    "sched_getaffinity",
):

    raise RuntimeError(
        "Linux CPU affinity API unavailable."
    )


available_affinity = sorted(
    os.sched_getaffinity(
        0
    )
)


print(
    "Notebook allowed CPU affinity:",
    available_affinity
)


if 0 not in available_affinity:

    raise RuntimeError(
        "Frozen CPU1 affinity CPU 0 is unavailable."
    )


taskset = shutil.which(
    "taskset"
)


print(
    "taskset:",
    taskset
)


if not taskset:

    raise RuntimeError(
        "taskset is required for frozen CPU1 child-process affinity."
    )


print(
    "Frozen CPU mode : CPU_1_PHYSICAL_CORE"
)

print(
    "Frozen affinity :",
    CPU_AFFINITY
)

print(
    "Frozen threads  :",
    THREAD_COUNT
)

print(
    "Worker timeout  :",
    WORKER_TIMEOUT_SECONDS,
    "seconds",
)


# =============================================================================
# 6. STATIC WORKER CLI RESOLUTION — BEFORE ANY TIMING
# =============================================================================

banner(
    "STAGE26-8C1 :: STATIC WORKER INVOCATION RESOLUTION"
)


def resolve_worker_invocation_style(
    worker_path,
):

    text = Path(
        worker_path
    ).read_text(
        encoding="utf-8"
    )


    # Historical worker style: config path from sys.argv[1].
    if re.search(
        r"sys\s*\.\s*argv\s*\[\s*1\s*\]",
        text,
    ):

        return {
            "style":
                "POSITIONAL_CONFIG_PATH",

            "build":
                lambda config_path: [
                    taskset,
                    "-c",
                    "0",
                    sys.executable,
                    str(
                        worker_path
                    ),
                    str(
                        config_path
                    ),
                ],
        }


    # Support an explicit --config parser only if statically visible.
    if (
        "argparse"
        in
        text
        and
        re.search(
            r"""add_argument\s*\(\s*["']--config["']""",
            text,
        )
    ):

        return {
            "style":
                "ARGPARSE_DASH_CONFIG",

            "build":
                lambda config_path: [
                    taskset,
                    "-c",
                    "0",
                    sys.executable,
                    str(
                        worker_path
                    ),
                    "--config",
                    str(
                        config_path
                    ),
                ],
        }


    raise RuntimeError(
        f"Worker invocation cannot be resolved statically without execution: "
        f"{worker_path}"
    )


v2_invocation = resolve_worker_invocation_style(
    V2_WORKER
)

v3_invocation = resolve_worker_invocation_style(
    V3_WORKER
)


print(
    "V2 invocation:",
    v2_invocation[
        "style"
    ]
)

print(
    "V3 invocation:",
    v3_invocation[
        "style"
    ]
)


if (
    v2_invocation[
        "style"
    ]
    !=
    v3_invocation[
        "style"
    ]
):

    raise RuntimeError(
        "V2/V3 invocation interface differs."
    )


# =============================================================================
# 7. LOAD EXACT HISTORICAL CONFIGS
# =============================================================================

banner(
    "STAGE26-8C1 :: EXACT HISTORICAL CONFIG SEMANTICS"
)


historical_configs = {}


for batch in BATCHES:

    candidates = sorted(
        C4_DIR.glob(
            f"conditions/batch_{batch}/*_config.json"
        )
    )


    if len(
        candidates
    ) != 1:

        raise RuntimeError(
            f"B={batch}: expected exactly one historical config."
        )


    path = candidates[
        0
    ]

    cfg = json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


    required = {
        "batch_size":
            batch,

        "start_index":
            26042,

        "end_index_exclusive":
            26042
            +
            batch,

        "affinity":
            [
                0
            ],

        "thread_count":
            1,

        "mode":
            "BENCHMARK_TIMED",

        "labels_allowed":
            False,

        "models_allowed":
            False,

        "gpu_allowed":
            False,

        "float32_div255_allowed":
            False,

        "corpus_dir":
            str(
                CORPUS_DIR
            ),
    }


    for key, expected in required.items():

        if cfg[
            key
        ] != expected:

            raise RuntimeError(
                f"B={batch}: historical config field changed: "
                f"{key}={cfg[key]!r}, expected={expected!r}"
            )


    historical_configs[
        batch
    ] = {
        "path":
            path,

        "sha256":
            sha256_file(
                path
            ),

        "config":
            cfg,
    }


    print(
        f"B={batch:5d} "
        f"warmup={cfg['warmup_runs']:3d} "
        f"timed={cfg['timed_runs']:3d} "
        f"SHA={historical_configs[batch]['sha256']}"
    )


# =============================================================================
# 8. CREATE TRANSIENT OUTPUT
# =============================================================================

OUT.mkdir(
    parents=True,
    exist_ok=False,
)

CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

LOG_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# =============================================================================
# 9. CHILD ENVIRONMENT
# =============================================================================

worker_env = os.environ.copy()


thread_env = {
    "OMP_NUM_THREADS":
        "1",

    "MKL_NUM_THREADS":
        "1",

    "OPENBLAS_NUM_THREADS":
        "1",

    "NUMEXPR_NUM_THREADS":
        "1",

    "VECLIB_MAXIMUM_THREADS":
        "1",
}


worker_env.update(
    thread_env
)


# Explicitly forbid CUDA visibility for this CPU-only sensitivity.
worker_env[
    "CUDA_VISIBLE_DEVICES"
] = ""


print(
    "\nChild thread environment:"
)


for key, value in thread_env.items():

    print(
        f"  {key}={value}"
    )


print(
    "  CUDA_VISIBLE_DEVICES=<empty>"
)


# =============================================================================
# 10. ENVIRONMENT GATE
# =============================================================================

def environment_gate():

    attempts = []


    for attempt in range(
        1,
        ENVIRONMENT_GATE_ATTEMPTS + 1,
    ):

        cpu_percent = float(
            psutil.cpu_percent(
                interval=1.0
            )
        )

        available_ram_bytes = int(
            psutil.virtual_memory().available
        )

        available_ram_gib = (
            available_ram_bytes
            /
            1024**3
        )


        passed = (
            cpu_percent
            <=
            MAX_CPU_PERCENT
            and
            available_ram_gib
            >=
            MIN_AVAILABLE_RAM_GIB
        )


        attempts.append(
            {
                "attempt":
                    attempt,

                "cpu_percent":
                    cpu_percent,

                "available_ram_bytes":
                    available_ram_bytes,

                "available_ram_gib":
                    available_ram_gib,

                "passed":
                    passed,
            }
        )


        print(
            f"    gate attempt {attempt}: "
            f"CPU={cpu_percent:.1f}% "
            f"RAM={available_ram_gib:.3f} GiB "
            f"{'PASS' if passed else 'FAIL'}"
        )


        if passed:

            return (
                True,
                attempts,
            )


        if attempt < ENVIRONMENT_GATE_ATTEMPTS:

            time.sleep(
                ENVIRONMENT_GATE_RETRY_SECONDS
            )


    return (
        False,
        attempts,
    )


# =============================================================================
# 11. RESULT SUMMARIZER
# =============================================================================

def summarize_worker_result(
    result,
    *,
    batch,
    expected_timed_runs,
):

    if result.get(
        "status"
    ) != "PASS":

        raise RuntimeError(
            f"Worker result status is not PASS: {result.get('status')!r}"
        )


    observations = result.get(
        "observations"
    )


    if not isinstance(
        observations,
        list,
    ):

        raise RuntimeError(
            "Worker result observations missing."
        )


    if len(
        observations
    ) != expected_timed_runs:

        raise RuntimeError(
            f"Observation count mismatch: "
            f"{len(observations)} != {expected_timed_runs}"
        )


    elapsed_ns = np.asarray(
        [
            int(
                row[
                    "elapsed_ns"
                ]
            )
            for row in observations
        ],
        dtype=np.int64,
    )


    if np.any(
        elapsed_ns
        <=
        0
    ):

        raise RuntimeError(
            "Non-positive elapsed_ns observation."
        )


    flows = np.asarray(
        [
            int(
                row[
                    "flows"
                ]
            )
            for row in observations
        ],
        dtype=np.int64,
    )


    if not np.all(
        flows
        ==
        batch
    ):

        raise RuntimeError(
            "Worker observation flow count differs from frozen batch."
        )


    throughput = np.asarray(
        [
            float(
                row[
                    "flows_per_second"
                ]
            )
            for row in observations
        ],
        dtype=np.float64,
    )


    if (
        np.any(
            ~np.isfinite(
                throughput
            )
        )
        or
        np.any(
            throughput
            <=
            0
        )
    ):

        raise RuntimeError(
            "Invalid flows_per_second observation."
        )


    # Frozen latency conversion policy:
    # float64 elapsed_ns -> ms before percentile.
    elapsed_ms = (
        elapsed_ns.astype(
            np.float64
        )
        /
        1_000_000.0
    )


    summary = {
        "n":
            int(
                len(
                    elapsed_ns
                )
            ),

        "p50_latency_ms":
            float(
                np.percentile(
                    elapsed_ms,
                    50,
                    method="linear",
                )
            ),

        "p95_latency_ms":
            float(
                np.percentile(
                    elapsed_ms,
                    95,
                    method="linear",
                )
            ),

        "p99_latency_ms":
            (
                float(
                    np.percentile(
                        elapsed_ms,
                        99,
                        method="linear",
                    )
                )
                if
                len(
                    elapsed_ns
                )
                >=
                100
                else
                None
            ),

        "median_throughput_flows_per_second":
            float(
                np.median(
                    throughput
                )
            ),

        "min_throughput_flows_per_second":
            float(
                np.min(
                    throughput
                )
            ),

        "max_throughput_flows_per_second":
            float(
                np.max(
                    throughput
                )
            ),

        "representation_fingerprint_sha256":
            result.get(
                "representation_fingerprint_sha256"
            ),

        "warm_representation_fingerprint_sha256":
            result.get(
                "warm_representation_fingerprint_sha256"
            ),
    }


    for key in [
        "p50_latency_ms",
        "p95_latency_ms",
        "median_throughput_flows_per_second",
    ]:

        finite_positive(
            summary[
                key
            ],
            key,
        )


    if summary[
        "p99_latency_ms"
    ] is not None:

        finite_positive(
            summary[
                "p99_latency_ms"
            ],
            "p99_latency_ms",
        )


    return (
        summary,
        observations,
    )


# =============================================================================
# 12. RUN ONE FRESH WORKER
# =============================================================================

worker_execution_rows = []

raw_observation_rows = []

worker_outputs = {}

worker_execution_counter = 0


def execute_worker(
    *,
    block,
    variant,
):

    global worker_execution_counter

    worker_execution_counter += 1


    block_index = int(
        block[
            "pair_block_execution_index"
        ]
    )

    batch = int(
        block[
            "batch_size"
        ]
    )

    pair_rep = int(
        block[
            "pair_rep"
        ]
    )


    if variant == "V2_ORIGINAL":

        worker_path = V2_WORKER

        worker_sha = EXPECTED_V2_SHA256

        invocation = v2_invocation

    elif variant == "V3_CLEAN":

        worker_path = V3_WORKER

        worker_sha = EXPECTED_V3_SHA256

        invocation = v3_invocation

    else:

        raise RuntimeError(
            f"Unknown frozen variant: {variant}"
        )


    historical = historical_configs[
        batch
    ]

    cfg = copy.deepcopy(
        historical[
            "config"
        ]
    )


    stem = (
        f"block_{block_index:02d}"
        f"__B{batch}"
        f"__pair_{pair_rep}"
        f"__{variant.lower()}"
    )


    config_path = (
        CONFIG_DIR
        /
        (
            stem
            +
            "_config.json"
        )
    )

    result_path = (
        RESULT_DIR
        /
        (
            stem
            +
            "_result.json"
        )
    )

    log_path = (
        LOG_DIR
        /
        (
            stem
            +
            ".log"
        )
    )


    # Only execution-specific fields change.
    # Historical timing/sample semantics remain exactly inherited.
    cfg[
        "result_path"
    ] = str(
        result_path
    )

    cfg[
        "worker_sha256"
    ] = worker_sha


    atomic_json(
        config_path,
        cfg,
    )


    config_sha = sha256_file(
        config_path
    )


    print(
        f"\n  WORKER {worker_execution_counter:02d}/50 "
        f"{variant}"
    )

    print(
        "    config SHA:",
        config_sha
    )


    gate_passed, gate_attempts = environment_gate()


    execution_row = {
        "worker_execution_index":
            worker_execution_counter,

        "pair_block_execution_index":
            block_index,

        "batch_size":
            batch,

        "pair_rep":
            pair_rep,

        "variant":
            variant,

        "worker_sha256":
            worker_sha,

        "historical_config_sha256":
            historical[
                "sha256"
            ],

        "execution_config_sha256":
            config_sha,

        "environment_gate_passed":
            gate_passed,

        "worker_status":
            None,

        "worker_return_code":
            None,

        "failure_class":
            None,

        "result_sha256":
            None,

        "n":
            None,

        "p50_latency_ms":
            None,

        "p95_latency_ms":
            None,

        "p99_latency_ms":
            None,

        "median_throughput_flows_per_second":
            None,

        "representation_fingerprint_sha256":
            None,
    }


    if not gate_passed:

        execution_row[
            "worker_status"
        ] = "ENVIRONMENT_GATE_FAILED"

        execution_row[
            "failure_class"
        ] = "ENVIRONMENT_GATE_RESOURCE_STATE"

        worker_execution_rows.append(
            execution_row
        )


        return {
            "status":
                "ENVIRONMENT_GATE_FAILED",

            "summary":
                None,

            "result":
                None,

            "gate_attempts":
                gate_attempts,

            "config_path":
                str(
                    config_path
                ),

            "log_path":
                None,
        }


    cmd = invocation[
        "build"
    ](
        config_path
    )


    print(
        "    fresh process: YES"
    )

    print(
        "    CPU affinity : [0]"
    )


    started_utc = datetime.now(
        timezone.utc
    ).isoformat()


    try:

        p = subprocess.run(
            cmd,
            cwd=REPO,
            env=worker_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=WORKER_TIMEOUT_SECONDS,
            check=False,
        )


        return_code = int(
            p.returncode
        )

        output_text = (
            p.stdout
            or
            ""
        )


        log_path.write_text(
            output_text,
            encoding="utf-8",
        )


        execution_row[
            "worker_return_code"
        ] = return_code


        print(
            "    return code  :",
            return_code
        )


        if return_code != 0:

            execution_row[
                "worker_status"
            ] = "WORKER_FAILED"

            execution_row[
                "failure_class"
            ] = "NONZERO_RETURN_CODE"

            worker_execution_rows.append(
                execution_row
            )


            print(
                "    status       : WORKER_FAILED"
            )


            if output_text.strip():

                print(
                    "    log tail:"
                )


                for line in output_text.splitlines()[
                    -20:
                ]:

                    print(
                        "      ",
                        line
                    )


            return {
                "status":
                    "WORKER_FAILED",

                "summary":
                    None,

                "result":
                    None,

                "gate_attempts":
                    gate_attempts,

                "config_path":
                    str(
                        config_path
                    ),

                "log_path":
                    str(
                        log_path
                    ),
            }


    except subprocess.TimeoutExpired as exc:

        output_text = (
            exc.stdout
            or
            ""
        )


        if isinstance(
            output_text,
            bytes,
        ):

            output_text = output_text.decode(
                "utf-8",
                errors="replace",
            )


        log_path.write_text(
            output_text,
            encoding="utf-8",
        )


        execution_row[
            "worker_status"
        ] = "TIMEOUT_RESOURCE_LIMIT"

        execution_row[
            "failure_class"
        ] = "TIMEOUT_RESOURCE_LIMIT"

        worker_execution_rows.append(
            execution_row
        )


        print(
            "    status       : TIMEOUT_RESOURCE_LIMIT"
        )


        return {
            "status":
                "TIMEOUT_RESOURCE_LIMIT",

            "summary":
                None,

            "result":
                None,

            "gate_attempts":
                gate_attempts,

            "config_path":
                str(
                    config_path
                ),

            "log_path":
                str(
                    log_path
                ),
        }


    finished_utc = datetime.now(
        timezone.utc
    ).isoformat()


    if not result_path.is_file():

        execution_row[
            "worker_status"
        ] = "RESULT_FILE_MISSING"

        execution_row[
            "failure_class"
        ] = "WORKER_OUTPUT_CONTRACT_FAILURE"

        worker_execution_rows.append(
            execution_row
        )


        print(
            "    status       : RESULT_FILE_MISSING"
        )


        return {
            "status":
                "RESULT_FILE_MISSING",

            "summary":
                None,

            "result":
                None,

            "gate_attempts":
                gate_attempts,

            "config_path":
                str(
                    config_path
                ),

            "log_path":
                str(
                    log_path
                ),
        }


    result_sha = sha256_file(
        result_path
    )


    result = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


    try:

        summary, observations = summarize_worker_result(
            result,
            batch=batch,
            expected_timed_runs=int(
                cfg[
                    "timed_runs"
                ]
            ),
        )


    except Exception as exc:

        execution_row[
            "worker_status"
        ] = "RESULT_VALIDATION_FAILED"

        execution_row[
            "failure_class"
        ] = (
            type(
                exc
            ).__name__
            +
            ": "
            +
            str(
                exc
            )
        )

        execution_row[
            "result_sha256"
        ] = result_sha

        worker_execution_rows.append(
            execution_row
        )


        print(
            "    status       : RESULT_VALIDATION_FAILED"
        )

        print(
            "    reason       :",
            repr(
                exc
            )
        )


        return {
            "status":
                "RESULT_VALIDATION_FAILED",

            "summary":
                None,

            "result":
                result,

            "gate_attempts":
                gate_attempts,

            "config_path":
                str(
                    config_path
                ),

            "result_path":
                str(
                    result_path
                ),

            "log_path":
                str(
                    log_path
                ),
        }


    execution_row.update(
        {
            "worker_status":
                "PASS",

            "result_sha256":
                result_sha,

            "n":
                summary[
                    "n"
                ],

            "p50_latency_ms":
                summary[
                    "p50_latency_ms"
                ],

            "p95_latency_ms":
                summary[
                    "p95_latency_ms"
                ],

            "p99_latency_ms":
                summary[
                    "p99_latency_ms"
                ],

            "median_throughput_flows_per_second":
                summary[
                    "median_throughput_flows_per_second"
                ],

            "representation_fingerprint_sha256":
                summary[
                    "representation_fingerprint_sha256"
                ],
        }
    )


    worker_execution_rows.append(
        execution_row
    )


    for observation in observations:

        raw_observation_rows.append(
            {
                "worker_execution_index":
                    worker_execution_counter,

                "pair_block_execution_index":
                    block_index,

                "batch_size":
                    batch,

                "pair_rep":
                    pair_rep,

                "variant":
                    variant,

                "iteration_index":
                    int(
                        observation[
                            "iteration_index"
                        ]
                    ),

                "elapsed_ns":
                    int(
                        observation[
                            "elapsed_ns"
                        ]
                    ),

                "elapsed_seconds":
                    float(
                        observation[
                            "elapsed_seconds"
                        ]
                    ),

                "flows":
                    int(
                        observation[
                            "flows"
                        ]
                    ),

                "flows_per_second":
                    float(
                        observation[
                            "flows_per_second"
                        ]
                    ),
            }
        )


    print(
        "    status       : PASS"
    )

    print(
        f"    p50          : "
        f"{summary['p50_latency_ms']:.6f} ms"
    )

    print(
        f"    p95          : "
        f"{summary['p95_latency_ms']:.6f} ms"
    )


    if summary[
        "p99_latency_ms"
    ] is None:

        print(
            "    p99          : N/A"
        )

    else:

        print(
            f"    p99          : "
            f"{summary['p99_latency_ms']:.6f} ms"
        )


    print(
        f"    median flows/s: "
        f"{summary['median_throughput_flows_per_second']:,.3f}"
    )

    print(
        "    fingerprint  :",
        summary[
            "representation_fingerprint_sha256"
        ]
    )


    return {
        "status":
            "PASS",

        "summary":
            summary,

        "result":
            result,

        "gate_attempts":
            gate_attempts,

        "started_utc":
            started_utc,

        "finished_utc":
            finished_utc,

        "config_path":
            str(
                config_path
            ),

        "result_path":
            str(
                result_path
            ),

        "log_path":
            str(
                log_path
            ),

        "result_sha256":
            result_sha,
    }


# =============================================================================
# 13. EXECUTE EXACT FROZEN PAIR ORDER
# =============================================================================

banner(
    "STAGE26-8C1 :: EXECUTE 25 FROZEN PAIR BLOCKS"
)


pair_result_rows = []


for block in pair_blocks:

    block_index = int(
        block[
            "pair_block_execution_index"
        ]
    )

    batch = int(
        block[
            "batch_size"
        ]
    )

    pair_rep = int(
        block[
            "pair_rep"
        ]
    )

    order = list(
        block[
            "within_pair_order"
        ]
    )


    print(
        "\n"
        +
        "-" * 124
    )

    print(
        f"PAIR BLOCK {block_index:02d}/25 "
        f":: B={batch} "
        f":: pair={pair_rep} "
        f":: order={order}"
    )

    print(
        "-" * 124
    )


    outputs = {}


    # Frozen within-pair order.
    for variant in order:

        outputs[
            variant
        ] = execute_worker(
            block=block,
            variant=variant,
        )


    worker_outputs[
        (
            batch,
            pair_rep,
        )
    ] = outputs


    v2 = outputs[
        "V2_ORIGINAL"
    ]

    v3 = outputs[
        "V3_CLEAN"
    ]


    complete = (
        v2[
            "status"
        ]
        ==
        "PASS"
        and
        v3[
            "status"
        ]
        ==
        "PASS"
    )


    pair_row = {
        "pair_block_execution_index":
            block_index,

        "batch_size":
            batch,

        "pair_rep":
            pair_rep,

        "within_pair_order":
            " -> ".join(
                order
            ),

        "V2_status":
            v2[
                "status"
            ],

        "V3_status":
            v3[
                "status"
            ],

        "pair_complete":
            complete,

        "V2_median_throughput_flows_per_second":
            None,

        "V3_median_throughput_flows_per_second":
            None,

        "throughput_ratio_clean_over_original":
            None,

        "V2_p50_latency_ms":
            None,

        "V3_p50_latency_ms":
            None,

        "p50_latency_ratio_clean_over_original":
            None,

        "V2_p95_latency_ms":
            None,

        "V3_p95_latency_ms":
            None,

        "p95_latency_ratio_clean_over_original":
            None,

        "V2_p99_latency_ms":
            None,

        "V3_p99_latency_ms":
            None,

        "p99_latency_ratio_clean_over_original":
            None,

        "V2_fingerprint":
            None,

        "V3_fingerprint":
            None,

        "fingerprint_equal":
            None,
    }


    if complete:

        s2 = v2[
            "summary"
        ]

        s3 = v3[
            "summary"
        ]


        fingerprint_equal = (
            s2[
                "representation_fingerprint_sha256"
            ]
            ==
            s3[
                "representation_fingerprint_sha256"
            ]
        )


        if not fingerprint_equal:

            raise RuntimeError(
                f"B={batch}/pair={pair_rep}: "
                "V2/V3 representation fingerprints differ."
            )


        throughput_ratio = (
            s3[
                "median_throughput_flows_per_second"
            ]
            /
            s2[
                "median_throughput_flows_per_second"
            ]
        )


        p50_ratio = (
            s3[
                "p50_latency_ms"
            ]
            /
            s2[
                "p50_latency_ms"
            ]
        )


        p95_ratio = (
            s3[
                "p95_latency_ms"
            ]
            /
            s2[
                "p95_latency_ms"
            ]
        )


        if (
            s2[
                "p99_latency_ms"
            ]
            is not None
            and
            s3[
                "p99_latency_ms"
            ]
            is not None
        ):

            p99_ratio = (
                s3[
                    "p99_latency_ms"
                ]
                /
                s2[
                    "p99_latency_ms"
                ]
            )

        else:

            p99_ratio = None


        pair_row.update(
            {
                "V2_median_throughput_flows_per_second":
                    s2[
                        "median_throughput_flows_per_second"
                    ],

                "V3_median_throughput_flows_per_second":
                    s3[
                        "median_throughput_flows_per_second"
                    ],

                "throughput_ratio_clean_over_original":
                    throughput_ratio,

                "V2_p50_latency_ms":
                    s2[
                        "p50_latency_ms"
                    ],

                "V3_p50_latency_ms":
                    s3[
                        "p50_latency_ms"
                    ],

                "p50_latency_ratio_clean_over_original":
                    p50_ratio,

                "V2_p95_latency_ms":
                    s2[
                        "p95_latency_ms"
                    ],

                "V3_p95_latency_ms":
                    s3[
                        "p95_latency_ms"
                    ],

                "p95_latency_ratio_clean_over_original":
                    p95_ratio,

                "V2_p99_latency_ms":
                    s2[
                        "p99_latency_ms"
                    ],

                "V3_p99_latency_ms":
                    s3[
                        "p99_latency_ms"
                    ],

                "p99_latency_ratio_clean_over_original":
                    p99_ratio,

                "V2_fingerprint":
                    s2[
                        "representation_fingerprint_sha256"
                    ],

                "V3_fingerprint":
                    s3[
                        "representation_fingerprint_sha256"
                    ],

                "fingerprint_equal":
                    fingerprint_equal,
            }
        )


        print(
            "\n  PAIR RESULT: COMPLETE"
        )

        print(
            f"    throughput V3/V2 : "
            f"{throughput_ratio:.9f}"
        )

        print(
            f"    p50 V3/V2        : "
            f"{p50_ratio:.9f}"
        )

        print(
            f"    p95 V3/V2        : "
            f"{p95_ratio:.9f}"
        )


        if p99_ratio is not None:

            print(
                f"    p99 V3/V2        : "
                f"{p99_ratio:.9f}"
            )

        else:

            print(
                "    p99 V3/V2        : N/A"
            )


        print(
            "    fingerprint equal:",
            fingerprint_equal
        )


    else:

        print(
            "\n  PAIR RESULT: INCOMPLETE — NO REPLACEMENT"
        )

        print(
            "    V2:",
            v2[
                "status"
            ]
        )

        print(
            "    V3:",
            v3[
                "status"
            ]
        )


    pair_result_rows.append(
        pair_row
    )


# =============================================================================
# 14. EXECUTION GEOMETRY
# =============================================================================

banner(
    "STAGE26-8C1 :: EXECUTION GEOMETRY"
)


print(
    "Frozen worker executions expected:",
    50
)

print(
    "Worker execution slots processed:",
    worker_execution_counter
)

print(
    "Execution rows:",
    len(
        worker_execution_rows
    )
)


if worker_execution_counter != 50:

    raise RuntimeError(
        "Did not process all 50 frozen worker-execution slots."
    )


if len(
    worker_execution_rows
) != 50:

    raise RuntimeError(
        "Worker execution receipt count != 50."
    )


worker_status_counts = Counter(
    row[
        "worker_status"
    ]
    for row in worker_execution_rows
)


print(
    "\nWorker status counts:"
)


for key in sorted(
    worker_status_counts
):

    print(
        f"  {key:34s}: "
        f"{worker_status_counts[key]}"
    )


complete_pairs = [
    row
    for row in pair_result_rows
    if row[
        "pair_complete"
    ]
]


print(
    "\nPair blocks expected :",
    25
)

print(
    "Pair rows           :",
    len(
        pair_result_rows
    )
)

print(
    "Complete pairs      :",
    len(
        complete_pairs
    )
)

print(
    "Incomplete pairs    :",
    25
    -
    len(
        complete_pairs
    )
)


if len(
    pair_result_rows
) != 25:

    raise RuntimeError(
        "Pair result row count != 25."
    )


# =============================================================================
# 15. BATCH-LEVEL FROZEN DESCRIPTIVE SUMMARIES
# =============================================================================

banner(
    "STAGE26-8C1 :: BATCH-LEVEL PAIRED SENSITIVITY SUMMARY"
)


batch_summary_rows = []


for batch in BATCHES:

    rows = [
        row
        for row in complete_pairs
        if int(
            row[
                "batch_size"
            ]
        )
        ==
        batch
    ]


    throughput_ratios = [
        float(
            row[
                "throughput_ratio_clean_over_original"
            ]
        )
        for row in rows
    ]

    p50_ratios = [
        float(
            row[
                "p50_latency_ratio_clean_over_original"
            ]
        )
        for row in rows
    ]

    p95_ratios = [
        float(
            row[
                "p95_latency_ratio_clean_over_original"
            ]
        )
        for row in rows
    ]

    p99_ratios = [
        float(
            row[
                "p99_latency_ratio_clean_over_original"
            ]
        )
        for row in rows
        if row[
            "p99_latency_ratio_clean_over_original"
        ]
        is not None
    ]


    def descriptor(
        values
    ):

        if not values:

            return {
                "median":
                    None,

                "min":
                    None,

                "max":
                    None,
            }


        arr = np.asarray(
            values,
            dtype=np.float64,
        )


        return {
            "median":
                float(
                    np.median(
                        arr
                    )
                ),

            "min":
                float(
                    np.min(
                        arr
                    )
                ),

            "max":
                float(
                    np.max(
                        arr
                    )
                ),
        }


    throughput_desc = descriptor(
        throughput_ratios
    )

    p50_desc = descriptor(
        p50_ratios
    )

    p95_desc = descriptor(
        p95_ratios
    )

    p99_desc = descriptor(
        p99_ratios
    )


    row = {
        "batch_size":
            batch,

        "frozen_pair_count":
            5,

        "complete_pair_count":
            len(
                rows
            ),

        "incomplete_pair_count":
            5
            -
            len(
                rows
            ),

        "throughput_ratio_pair_values":
            json.dumps(
                throughput_ratios
            ),

        "throughput_ratio_median":
            throughput_desc[
                "median"
            ],

        "throughput_ratio_min":
            throughput_desc[
                "min"
            ],

        "throughput_ratio_max":
            throughput_desc[
                "max"
            ],

        "p50_ratio_pair_values":
            json.dumps(
                p50_ratios
            ),

        "p50_ratio_median":
            p50_desc[
                "median"
            ],

        "p50_ratio_min":
            p50_desc[
                "min"
            ],

        "p50_ratio_max":
            p50_desc[
                "max"
            ],

        "p95_ratio_pair_values":
            json.dumps(
                p95_ratios
            ),

        "p95_ratio_median":
            p95_desc[
                "median"
            ],

        "p95_ratio_min":
            p95_desc[
                "min"
            ],

        "p95_ratio_max":
            p95_desc[
                "max"
            ],

        "p99_ratio_pair_values":
            (
                json.dumps(
                    p99_ratios
                )
                if p99_ratios
                else
                ""
            ),

        "p99_ratio_median":
            p99_desc[
                "median"
            ],

        "p99_ratio_min":
            p99_desc[
                "min"
            ],

        "p99_ratio_max":
            p99_desc[
                "max"
            ],

        "bootstrap":
            False,

        "hypothesis_test":
            False,

        "materiality_threshold":
            "",
    }


    batch_summary_rows.append(
        row
    )


    print(
        f"\nB={batch}"
    )

    print(
        f"  complete pairs : "
        f"{len(rows)}/5"
    )


    if throughput_ratios:

        print(
            "  throughput V3/V2 pair ratios:"
        )

        print(
            "   ",
            [
                round(
                    x,
                    9,
                )
                for x in throughput_ratios
            ]
        )

        print(
            f"  throughput median/min/max : "
            f"{throughput_desc['median']:.9f} / "
            f"{throughput_desc['min']:.9f} / "
            f"{throughput_desc['max']:.9f}"
        )

        print(
            f"  p50 median V3/V2          : "
            f"{p50_desc['median']:.9f}"
        )

        print(
            f"  p95 median V3/V2          : "
            f"{p95_desc['median']:.9f}"
        )


        if p99_desc[
            "median"
        ] is not None:

            print(
                f"  p99 median V3/V2          : "
                f"{p99_desc['median']:.9f}"
            )

        else:

            print(
                "  p99 median V3/V2          : N/A"
            )


    else:

        print(
            "  no complete frozen pair available"
        )


# =============================================================================
# 16. WRITE RAW / PAIR / SUMMARY TABLES
# =============================================================================

banner(
    "STAGE26-8C1 :: WRITE TRANSIENT RESULT PACKAGE"
)


worker_fields = [
    "worker_execution_index",
    "pair_block_execution_index",
    "batch_size",
    "pair_rep",
    "variant",
    "worker_sha256",
    "historical_config_sha256",
    "execution_config_sha256",
    "environment_gate_passed",
    "worker_status",
    "worker_return_code",
    "failure_class",
    "result_sha256",
    "n",
    "p50_latency_ms",
    "p95_latency_ms",
    "p99_latency_ms",
    "median_throughput_flows_per_second",
    "representation_fingerprint_sha256",
]


write_csv(
    WORKER_EXECUTION_CSV,
    worker_execution_rows,
    worker_fields,
)


raw_fields = [
    "worker_execution_index",
    "pair_block_execution_index",
    "batch_size",
    "pair_rep",
    "variant",
    "iteration_index",
    "elapsed_ns",
    "elapsed_seconds",
    "flows",
    "flows_per_second",
]


write_csv(
    RAW_OBSERVATIONS_CSV,
    raw_observation_rows,
    raw_fields,
)


pair_fields = [
    "pair_block_execution_index",
    "batch_size",
    "pair_rep",
    "within_pair_order",
    "V2_status",
    "V3_status",
    "pair_complete",
    "V2_median_throughput_flows_per_second",
    "V3_median_throughput_flows_per_second",
    "throughput_ratio_clean_over_original",
    "V2_p50_latency_ms",
    "V3_p50_latency_ms",
    "p50_latency_ratio_clean_over_original",
    "V2_p95_latency_ms",
    "V3_p95_latency_ms",
    "p95_latency_ratio_clean_over_original",
    "V2_p99_latency_ms",
    "V3_p99_latency_ms",
    "p99_latency_ratio_clean_over_original",
    "V2_fingerprint",
    "V3_fingerprint",
    "fingerprint_equal",
]


write_csv(
    PAIR_RESULTS_CSV,
    pair_result_rows,
    pair_fields,
)


batch_fields = list(
    batch_summary_rows[
        0
    ].keys()
)


write_csv(
    BATCH_SUMMARY_CSV,
    batch_summary_rows,
    batch_fields,
)


# =============================================================================
# 17. RESULTS JSON
# =============================================================================

results_payload = {
    "schema":
        "stage26_8c1_paired_representation_sensitivity_results_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "status":
        (
            "PASS_COMPLETE_FROZEN_PAIRED_SENSITIVITY"
            if
            len(
                complete_pairs
            )
            ==
            25
            else
            "COMPLETE_EXECUTION_WITH_INCOMPLETE_FROZEN_PAIRS"
        ),

    "design": {
        "type":
            "PAIRED_CONTEMPORANEOUS_A_B_SENSITIVITY",

        "pair_blocks":
            25,

        "worker_execution_slots":
            50,

        "complete_pairs":
            len(
                complete_pairs
            ),

        "incomplete_pairs":
            25
            -
            len(
                complete_pairs
            ),

        "CPU_mode":
            "CPU_1_PHYSICAL_CORE",

        "affinity":
            [
                0
            ],

        "thread_count":
            1,

        "batches":
            BATCHES,

        "bootstrap":
            False,

        "hypothesis_test":
            False,

        "materiality_threshold":
            None,

        "replacement_pairs":
            False,
    },

    "worker_status_counts":
        dict(
            worker_status_counts
        ),

    "pair_results":
        pair_result_rows,

    "batch_summaries":
        batch_summary_rows,

    "interpretation_boundary": {
        "role":
            "DESCRIPTIVE_PAIRED_SENSITIVITY_ONLY",

        "primary_estimand":
            (
                "median_throughput_V3 / median_throughput_V2"
            ),

        "V3_replaces_historical_4C3":
            False,

        "historical_4C3_modified":
            False,

        "Stage26_7C_recomputed":
            False,

        "full_pipeline_claim":
            False,

        "Pareto_recomputed":
            False,
    },
}


atomic_json(
    RESULTS_JSON,
    results_payload,
)


# =============================================================================
# 18. EXECUTION RECEIPT
# =============================================================================

execution_receipt = {
    "schema":
        "stage26_8c1_execution_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "status":
        (
            "PASS_ALL_25_PAIRS_COMPLETE"
            if
            len(
                complete_pairs
            )
            ==
            25
            else
            "PASS_FROZEN_PLAN_EXECUTED_WITH_REPORTED_INCOMPLETE_PAIRS"
        ),

    "frozen_identity": {
        "protocol_sha256":
            EXPECTED_PROTOCOL_SHA256,

        "execution_plan_sha256":
            EXPECTED_EXECUTION_PLAN_SHA256,

        "V2_worker_sha256":
            EXPECTED_V2_SHA256,

        "V3_worker_sha256":
            EXPECTED_V3_SHA256,

        "freeze_receipt_sha256":
            EXPECTED_FREEZE_RECEIPT_SHA256,

        "lock_manifest_sha256":
            EXPECTED_LOCK_MANIFEST_SHA256,

        "restore_receipt_sha256":
            EXPECTED_RESTORE_RECEIPT_SHA256,
    },

    "execution": {
        "pair_blocks_processed":
            25,

        "worker_execution_slots_processed":
            worker_execution_counter,

        "worker_status_counts":
            dict(
                worker_status_counts
            ),

        "complete_pairs":
            len(
                complete_pairs
            ),

        "incomplete_pairs":
            25
            -
            len(
                complete_pairs
            ),

        "raw_observations":
            len(
                raw_observation_rows
            ),

        "replacement_pairs":
            0,

        "adaptive_reordering":
            False,
    },

    "outputs": {
        "worker_executions_csv_sha256":
            sha256_file(
                WORKER_EXECUTION_CSV
            ),

        "raw_observations_csv_sha256":
            sha256_file(
                RAW_OBSERVATIONS_CSV
            ),

        "pair_results_csv_sha256":
            sha256_file(
                PAIR_RESULTS_CSV
            ),

        "batch_summary_csv_sha256":
            sha256_file(
                BATCH_SUMMARY_CSV
            ),

        "results_json_sha256":
            sha256_file(
                RESULTS_JSON
            ),
    },

    "scientific_state": {
        "sensitivity_timing_executed":
            True,

        "historical_4C3_modified":
            False,

        "historical_4C3_replaced":
            False,

        "Stage26_7C_recomputed":
            False,

        "bootstrap_executed":
            False,

        "hypothesis_test_performed":
            False,

        "materiality_threshold_selected":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "Pareto_recomputed":
            False,

        "PCAP_accessed":
            False,

        "labels_accessed":
            False,

        "GPU_used":
            False,

        "Git_modified":
            False,
    },

    "next":
        (
            "Inspect the complete frozen paired sensitivity result, then "
            "durably anchor the unchanged raw evidence and descriptive "
            "summaries before Stage26 CPU closure."
        ),
}


atomic_json(
    EXECUTION_RECEIPT,
    execution_receipt,
)


# =============================================================================
# 19. PACKAGE HASHES
# =============================================================================

banner(
    "STAGE26-8C1 :: TRANSIENT PACKAGE HASHES"
)


package_paths = [
    WORKER_EXECUTION_CSV,
    RAW_OBSERVATIONS_CSV,
    PAIR_RESULTS_CSV,
    BATCH_SUMMARY_CSV,
    RESULTS_JSON,
    EXECUTION_RECEIPT,
]


for path in package_paths:

    print(
        f"{path.name:52s} "
        f"{path.stat().st_size:10,d} B "
        f"{sha256_file(path)}"
    )


# =============================================================================
# 20. FINAL SCIENTIFIC SUMMARY
# =============================================================================

banner(
    "STAGE26-8C1 PAIRED SENSITIVITY EXECUTION COMPLETE"
)


print(
    "Worker execution slots:",
    worker_execution_counter,
    "/ 50"
)

print(
    "Complete pairs:",
    len(
        complete_pairs
    ),
    "/ 25"
)

print(
    "Incomplete pairs:",
    25
    -
    len(
        complete_pairs
    )
)


print(
    "\nBATCH SUMMARY:"
)


for row in batch_summary_rows:

    batch = int(
        row[
            "batch_size"
        ]
    )

    count = int(
        row[
            "complete_pair_count"
        ]
    )


    print(
        f"\n  B={batch}"
    )

    print(
        f"    complete pairs : {count}/5"
    )


    if row[
        "throughput_ratio_median"
    ] is not None:

        print(
            f"    throughput V3/V2 median : "
            f"{row['throughput_ratio_median']:.9f}"
        )

        print(
            f"    throughput V3/V2 range  : "
            f"{row['throughput_ratio_min']:.9f}"
            f" .. "
            f"{row['throughput_ratio_max']:.9f}"
        )

        print(
            f"    p50 V3/V2 median        : "
            f"{row['p50_ratio_median']:.9f}"
        )

        print(
            f"    p95 V3/V2 median        : "
            f"{row['p95_ratio_median']:.9f}"
        )


        if row[
            "p99_ratio_median"
        ] is not None:

            print(
                f"    p99 V3/V2 median        : "
                f"{row['p99_ratio_median']:.9f}"
            )

        else:

            print(
                "    p99 V3/V2 median        : N/A"
            )


print(
    "\nINTERPRETATION:"
)

print(
    "  descriptive paired sensitivity only"
)

print(
    "  no significance claim"
)

print(
    "  no materiality threshold"
)

print(
    "  no bootstrap"
)

print(
    "  original Stage26-4C3 retained"
)

print(
    "  V3 is NOT a replacement measurement"
)

print(
    "  Stage26-7C remains unchanged"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  sensitivity timing      : YES"
)

print(
    "  historical 4C3 modified : NO"
)

print(
    "  Stage26-7C recomputed   : NO"
)

print(
    "  model inference          : NO"
)

print(
    "  bootstrap                : NO"
)

print(
    "  hypothesis test          : NO"
)

print(
    "  Pareto                   : NO"
)

print(
    "  PCAP                     : NO"
)

print(
    "  labels                   : NO"
)

print(
    "  GPU                      : NO"
)


# =============================================================================
# 21. FINAL GIT GATE
# =============================================================================

final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "\nGIT:"
)

print(
    "  HEAD       :",
    final_head
)

print(
    "  origin/main:",
    final_remote
)

print(
    "  repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed during sensitivity execution."
    )


if final_remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed during sensitivity execution."
    )


if final_status:

    raise RuntimeError(
        "Sensitivity execution unexpectedly modified Git."
    )


print(
    "\nTRANSIENT OUTPUT:"
)

print(
    " ",
    OUT
)


print(
    "\nNEXT:"
)

print(
    "  Inspect these frozen paired sensitivity results."
)

print(
    "  Do NOT rerun or replace any pair based on the observed values."
)

print(
    "  If the execution package is internally valid, the next cell will"
)

print(
    "  durably commit/push the exact raw evidence and descriptive results."
)


STAGE26-8C1 :: DURABLE SCIENTIFIC STATE
Expected parent: 973e0c479ce91ad21e92ed5db11585f26a4c49ec
Local HEAD     : 973e0c479ce91ad21e92ed5db11585f26a4c49ec
origin/main    : 973e0c479ce91ad21e92ed5db11585f26a4c49ec
Repo clean     : True

STAGE26-8C1 :: EXACT FROZEN SENSITIVITY LOCK
protocol           PASS 4ce96c1540b29bd9683305ed981412b46be4b0b2596dcb58b3064f0724f05c5e
execution plan     PASS 5581bb2576919930913f49a308470083e7b27e315bc5cd99ba1fc1a853dcac14
freeze receipt     PASS d052bcd8bf3b2ed6a93f1ca3813036c030fa1d1de1fb28ffea166cf3e226bc33
lock manifest      PASS 6cfa4790ccabd1d755af40960b2e38846b4e15fe92ba2fac70d72ac3c162334f
V2 worker          PASS ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23
V3 worker          PASS c065e70fc16a628e482961f0db669d95affb841edd39153abf28dc99e868ad02

Pair blocks      : 25
Worker processes : 50
Batches          : [1, 64, 256, 1024, 8192]
Randomization    : FROZEN

STAGE26-8C1 :: RESTORED INPUT GATE
Restore receipt SHA256:
  babcf2

In [11]:
# =============================================================================
# STAGE26-8C2
# AUDIT + DURABLY ANCHOR EXACT STAGE26-8C1 PAIRED SENSITIVITY EVIDENCE
#
# DURABLE SCIENTIFIC PARENT:
#   973e0c479ce91ad21e92ed5db11585f26a4c49ec
#
# PURPOSE
# -------
# Stage26-8C1 has already executed the prospectively frozen paired experiment.
#
# Observed execution geometry:
#
#   worker executions : 50 / 50 PASS
#   frozen pairs       : 25 / 25 complete
#   incomplete pairs   : 0
#   replacements       : 0
#
# This cell DOES NOT rerun any timing.
#
# It:
#   1. verifies exact hashes of the six Stage26-8C1 top-level outputs;
#   2. verifies the exact Stage26-8B protocol/worker locks;
#   3. verifies the exact Stage26-8C0 source-restoration receipt;
#   4. audits execution geometry against the frozen randomized plan;
#   5. verifies 50 execution configs, 50 raw worker result JSONs and 50 logs;
#   6. verifies every reported result/config hash against the worker table;
#   7. verifies all 25 V2/V3 representation fingerprints are equal;
#   8. verifies the frozen descriptive batch summaries;
#   9. copies the raw evidence BYTE-FOR-BYTE into Git;
#  10. records a descriptive audit interpretation without adding a
#      hypothesis test, bootstrap, or materiality threshold;
#  11. commits, pushes and remotely byte-verifies the package.
#
# IMPORTANT SCIENTIFIC INTERPRETATION
# -----------------------------------
# The sensitivity direction is batch-dependent:
#
#   B=1     throughput V3/V2 median = 1.044551992
#   B=64    throughput V3/V2 median = 0.886861376
#   B=256   throughput V3/V2 median = 0.933557505
#   B=1024  throughput V3/V2 median = 0.971043122
#   B=8192  throughput V3/V2 median = 1.004347747
#
# Therefore:
#
#   - historical Stage26-4C3 remains the ORIGINAL measurement;
#   - V3 is NOT promoted to a replacement measurement;
#   - Stage26-7C remains unchanged;
#   - the previously open 4C3 implementation-sensitivity audit can be
#     considered COMPLETED once this evidence is remotely durable;
#   - final publication wording must explicitly distinguish historical
#     4C3 from this paired robustness/sensitivity evidence.
#
# NO:
#   - representation execution
#   - timing
#   - pair replacement
#   - new ratio experiment
#   - bootstrap
#   - hypothesis test
#   - significance testing
#   - materiality threshold
#   - Stage26-7C recomputation
#   - historical 4C3 replacement
#   - inference
#   - Pareto recomputation
#   - PCAP
#   - labels
#   - GPU
# =============================================================================

from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import shutil
import stat
import subprocess
import tempfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "973e0c479ce91ad21e92ed5db11585f26a4c49ec"
)

COMMIT_SUBJECT = (
    "stage26: anchor paired representation sensitivity results"
)


# -----------------------------------------------------------------------------
# Stage26-8B frozen lock
# -----------------------------------------------------------------------------

LOCK_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_8b_representation_sensitivity_protocol_lock"
)

PROTOCOL = (
    LOCK_DIR
    / "stage26_8b_representation_sensitivity_protocol.json"
)

EXECUTION_PLAN = (
    LOCK_DIR
    / "stage26_8b_representation_sensitivity_execution_plan.json"
)

FREEZE_RECEIPT = (
    LOCK_DIR
    / "stage26_8b_representation_sensitivity_freeze_receipt.json"
)

LOCK_MANIFEST = (
    LOCK_DIR
    / "stage26_8b_representation_sensitivity_lock_manifest.json"
)

V3_WORKER = (
    LOCK_DIR
    / "stage26_representation_worker_v3_sensitivity.py"
)

V2_WORKER = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_4c1a_label_free_equivalence_erratum"
    / "stage26_representation_worker_v2.py"
)


EXPECTED_PROTOCOL_SHA256 = (
    "4ce96c1540b29bd9683305ed981412b46be4b0b2596dcb58b3064f0724f05c5e"
)

EXPECTED_EXECUTION_PLAN_SHA256 = (
    "5581bb2576919930913f49a308470083e7b27e315bc5cd99ba1fc1a853dcac14"
)

EXPECTED_FREEZE_RECEIPT_SHA256 = (
    "d052bcd8bf3b2ed6a93f1ca3813036c030fa1d1de1fb28ffea166cf3e226bc33"
)

EXPECTED_LOCK_MANIFEST_SHA256 = (
    "6cfa4790ccabd1d755af40960b2e38846b4e15fe92ba2fac70d72ac3c162334f"
)

EXPECTED_V2_SHA256 = (
    "ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23"
)

EXPECTED_V3_SHA256 = (
    "c065e70fc16a628e482961f0db669d95affb841edd39153abf28dc99e868ad02"
)


# -----------------------------------------------------------------------------
# Stage26-8C0 exact input restoration
# -----------------------------------------------------------------------------

RUNTIME_ROOT = Path(
    "/kaggle/working/stage26_deployment_profiling"
)

RESTORE_RECEIPT = (
    RUNTIME_ROOT
    / "cpu_closure"
    / "stage26_8c0_compact_input_restore_receipt.json"
)

EXPECTED_RESTORE_RECEIPT_SHA256 = (
    "babcf2331db7c242297b346bad0904bb061251acb83003b5222c16295de96ff1"
)


# -----------------------------------------------------------------------------
# Stage26-8C1 transient measurement package
# -----------------------------------------------------------------------------

TRANSIENT = (
    RUNTIME_ROOT
    / "cpu_closure"
    / "stage26_8c1_paired_representation_sensitivity"
)

TRANSIENT_CONFIG_DIR = (
    TRANSIENT
    / "configs"
)

TRANSIENT_RESULT_DIR = (
    TRANSIENT
    / "worker_results"
)

TRANSIENT_LOG_DIR = (
    TRANSIENT
    / "worker_logs"
)

WORKER_EXECUTION_CSV = (
    TRANSIENT
    / "stage26_8c1_worker_executions.csv"
)

RAW_OBSERVATIONS_CSV = (
    TRANSIENT
    / "stage26_8c1_raw_observations.csv"
)

PAIR_RESULTS_CSV = (
    TRANSIENT
    / "stage26_8c1_pair_results.csv"
)

BATCH_SUMMARY_CSV = (
    TRANSIENT
    / "stage26_8c1_batch_sensitivity_summary.csv"
)

RESULTS_JSON = (
    TRANSIENT
    / "stage26_8c1_paired_sensitivity_results.json"
)

EXECUTION_RECEIPT = (
    TRANSIENT
    / "stage26_8c1_execution_receipt.json"
)


# Exact hashes printed by completed Cell 49.
EXPECTED_TOP_LEVEL_HASHES = {
    "stage26_8c1_worker_executions.csv":
        "c2e633afd44c93d703f97756d1c393767c4efda4b1329599f34c97741b23d884",

    "stage26_8c1_raw_observations.csv":
        "2a313e20430069d6c253438edb9653bcc34aa52c81800e6cb7ff84af4bf48bec",

    "stage26_8c1_pair_results.csv":
        "d1b8b751dc89f934e520683c5a81a8fed4d990d1f3d3304e4c73bed100e56048",

    "stage26_8c1_batch_sensitivity_summary.csv":
        "3d00f76cb7f3d9dd13069f79ecfe740e6fa7d1718e035dc788f8ad8d709b424e",

    "stage26_8c1_paired_sensitivity_results.json":
        "17d80752b38e13b3cc65dc1791fe0353765edd4a71798613ab878a0cdd835312",

    "stage26_8c1_execution_receipt.json":
        "4cb2292b1526cedb60caec07a141372471abc7e5ad5475a01affb1174b7ba978",
}


# Frozen descriptive results observed in completed Stage26-8C1.
EXPECTED_BATCH_MEDIANS = {
    1: {
        "throughput":
            1.044551992,

        "p50":
            0.957348072,

        "p95":
            0.923291605,

        "p99":
            1.034319340,
    },

    64: {
        "throughput":
            0.886861376,

        "p50":
            1.127571744,

        "p95":
            1.118754262,

        "p99":
            1.087208735,
    },

    256: {
        "throughput":
            0.933557505,

        "p50":
            1.071497588,

        "p95":
            1.068124629,

        "p99":
            1.051769766,
    },

    1024: {
        "throughput":
            0.971043122,

        "p50":
            1.030081286,

        "p95":
            1.080270946,

        "p99":
            None,
    },

    8192: {
        "throughput":
            1.004347747,

        "p50":
            0.995671484,

        "p95":
            0.975563952,

        "p99":
            None,
    },
}


BATCHES = [
    1,
    64,
    256,
    1024,
    8192,
]

EXPECTED_RAW_OBSERVATIONS = (
    5200
)


# -----------------------------------------------------------------------------
# Durable result checkpoint
# -----------------------------------------------------------------------------

CHECKPOINT_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_8c1_paired_representation_sensitivity"
)

CHECKPOINT = (
    REPO
    / CHECKPOINT_REL
)

DURABLE_EVIDENCE_DIR = (
    CHECKPOINT
    / "evidence"
)

DURABLE_CONFIG_DIR = (
    DURABLE_EVIDENCE_DIR
    / "configs"
)

DURABLE_RESULT_DIR = (
    DURABLE_EVIDENCE_DIR
    / "worker_results"
)

DURABLE_LOG_DIR = (
    DURABLE_EVIDENCE_DIR
    / "worker_logs"
)

DURABLE_PROVENANCE_DIR = (
    CHECKPOINT
    / "provenance"
)

DURABLE_RESTORE_RECEIPT = (
    DURABLE_PROVENANCE_DIR
    / "stage26_8c0_compact_input_restore_receipt.json"
)

AUDIT_JSON = (
    CHECKPOINT
    / "stage26_8c1_sensitivity_audit.json"
)

ANCHOR_RECEIPT = (
    CHECKPOINT
    / "stage26_8c2_anchor_receipt.json"
)

MANIFEST = (
    CHECKPOINT
    / "stage26_8c1_paired_sensitivity_manifest.json"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    env=None,
    check=True,
    text=True,
):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        check=False,
    )

    if check and p.returncode != 0:

        output = (
            p.stdout
            if text
            else
            p.stdout.decode(
                "utf-8",
                errors="replace",
            )
        )

        raise RuntimeError(
            "$ "
            +
            " ".join(
                map(
                    str,
                    cmd,
                )
            )
            +
            "\n\n"
            +
            output
        )

    return p


def git(
    *args,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
    ).stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_json(
    path,
    payload,
):

    path = Path(
        path
    )

    tmp = Path(
        str(path)
        +
        ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )


    os.replace(
        tmp,
        path,
    )


def read_csv(path):

    with Path(path).open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def bool_csv(value):

    if value == "True":
        return True

    if value == "False":
        return False

    raise RuntimeError(
        f"Expected CSV boolean; got {value!r}"
    )


def close_float(
    actual,
    expected,
    *,
    atol=5e-10,
):

    return math.isclose(
        float(
            actual
        ),
        float(
            expected
        ),
        rel_tol=0.0,
        abs_tol=atol,
    )


def git_blob_bytes(
    treeish,
    repo_relative_path,
):

    p = run(
        [
            "git",
            "show",
            f"{treeish}:{repo_relative_path}",
        ],
        text=False,
    )

    return bytes(
        p.stdout
    )


# =============================================================================
# 2. DURABLE PARENT GATE
# =============================================================================

banner(
    "STAGE26-8C2 :: DURABLE SCIENTIFIC PARENT"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-8C2 parent."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Stage26-8C2."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if CHECKPOINT.exists():

    raise RuntimeError(
        "Stage26-8C1 durable checkpoint already exists."
    )


# =============================================================================
# 3. EXACT UPSTREAM LOCK / RESTORE IDENTITIES
# =============================================================================

banner(
    "STAGE26-8C2 :: UPSTREAM IDENTITY"
)


upstream_checks = [
    (
        "protocol",
        PROTOCOL,
        EXPECTED_PROTOCOL_SHA256,
    ),

    (
        "execution plan",
        EXECUTION_PLAN,
        EXPECTED_EXECUTION_PLAN_SHA256,
    ),

    (
        "freeze receipt",
        FREEZE_RECEIPT,
        EXPECTED_FREEZE_RECEIPT_SHA256,
    ),

    (
        "lock manifest",
        LOCK_MANIFEST,
        EXPECTED_LOCK_MANIFEST_SHA256,
    ),

    (
        "V2 worker",
        V2_WORKER,
        EXPECTED_V2_SHA256,
    ),

    (
        "V3 worker",
        V3_WORKER,
        EXPECTED_V3_SHA256,
    ),

    (
        "restore receipt",
        RESTORE_RECEIPT,
        EXPECTED_RESTORE_RECEIPT_SHA256,
    ),
]


for label, path, expected in upstream_checks:

    if not Path(path).is_file():

        raise FileNotFoundError(
            path
        )


    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:18s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Upstream identity mismatch: {label}"
        )


# =============================================================================
# 4. EXACT STAGE26-8C1 TOP-LEVEL HASH GATE
# =============================================================================

banner(
    "STAGE26-8C2 :: EXACT COMPLETED-MEASUREMENT HASH GATE"
)


top_level_paths = {
    WORKER_EXECUTION_CSV.name:
        WORKER_EXECUTION_CSV,

    RAW_OBSERVATIONS_CSV.name:
        RAW_OBSERVATIONS_CSV,

    PAIR_RESULTS_CSV.name:
        PAIR_RESULTS_CSV,

    BATCH_SUMMARY_CSV.name:
        BATCH_SUMMARY_CSV,

    RESULTS_JSON.name:
        RESULTS_JSON,

    EXECUTION_RECEIPT.name:
        EXECUTION_RECEIPT,
}


for filename, expected_sha in EXPECTED_TOP_LEVEL_HASHES.items():

    path = top_level_paths[
        filename
    ]


    if not path.is_file():

        raise FileNotFoundError(
            path
        )


    actual_sha = sha256_file(
        path
    )

    passed = (
        actual_sha
        ==
        expected_sha
    )


    print(
        f"{filename:52s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual_sha}"
    )


    if not passed:

        raise RuntimeError(
            f"Completed Stage26-8C1 output changed: {filename}"
        )


# =============================================================================
# 5. READ EXACT FROZEN PLAN + RESULT TABLES
# =============================================================================

banner(
    "STAGE26-8C2 :: EXECUTION GEOMETRY AUDIT"
)


plan = json.loads(
    EXECUTION_PLAN.read_text(
        encoding="utf-8"
    )
)

protocol = json.loads(
    PROTOCOL.read_text(
        encoding="utf-8"
    )
)

execution_receipt = json.loads(
    EXECUTION_RECEIPT.read_text(
        encoding="utf-8"
    )
)

results = json.loads(
    RESULTS_JSON.read_text(
        encoding="utf-8"
    )
)


worker_rows = read_csv(
    WORKER_EXECUTION_CSV
)

raw_rows = read_csv(
    RAW_OBSERVATIONS_CSV
)

pair_rows = read_csv(
    PAIR_RESULTS_CSV
)

batch_rows = read_csv(
    BATCH_SUMMARY_CSV
)


print(
    "worker rows:",
    len(
        worker_rows
    )
)

print(
    "raw observations:",
    len(
        raw_rows
    )
)

print(
    "pair rows:",
    len(
        pair_rows
    )
)

print(
    "batch rows:",
    len(
        batch_rows
    )
)


if len(
    worker_rows
) != 50:

    raise RuntimeError(
        "Expected exactly 50 worker rows."
    )


if len(
    raw_rows
) != EXPECTED_RAW_OBSERVATIONS:

    raise RuntimeError(
        f"Expected {EXPECTED_RAW_OBSERVATIONS} raw observations."
    )


if len(
    pair_rows
) != 25:

    raise RuntimeError(
        "Expected exactly 25 pair rows."
    )


if len(
    batch_rows
) != 5:

    raise RuntimeError(
        "Expected exactly five batch summary rows."
    )


if execution_receipt[
    "status"
] != "PASS_ALL_25_PAIRS_COMPLETE":

    raise RuntimeError(
        "Execution receipt is not all-pairs-complete."
    )


if results[
    "status"
] != "PASS_COMPLETE_FROZEN_PAIRED_SENSITIVITY":

    raise RuntimeError(
        "Result package not marked fully complete."
    )


if execution_receipt[
    "execution"
][
    "worker_execution_slots_processed"
] != 50:

    raise RuntimeError(
        "Execution receipt worker count mismatch."
    )


if execution_receipt[
    "execution"
][
    "complete_pairs"
] != 25:

    raise RuntimeError(
        "Execution receipt pair count mismatch."
    )


if execution_receipt[
    "execution"
][
    "incomplete_pairs"
] != 0:

    raise RuntimeError(
        "Unexpected incomplete pair."
    )


if execution_receipt[
    "execution"
][
    "replacement_pairs"
] != 0:

    raise RuntimeError(
        "Replacement pair detected."
    )


if execution_receipt[
    "execution"
][
    "adaptive_reordering"
] is not False:

    raise RuntimeError(
        "Adaptive reordering detected."
    )


# =============================================================================
# 6. EXACT FROZEN EXECUTION-ORDER AUDIT
# =============================================================================

banner(
    "STAGE26-8C2 :: FROZEN RANDOMIZED ORDER AUDIT"
)


expected_execution_sequence = []


for block in plan[
    "pair_blocks"
]:

    block_index = int(
        block[
            "pair_block_execution_index"
        ]
    )

    batch = int(
        block[
            "batch_size"
        ]
    )

    pair_rep = int(
        block[
            "pair_rep"
        ]
    )


    for variant in block[
        "within_pair_order"
    ]:

        expected_execution_sequence.append(
            (
                block_index,
                batch,
                pair_rep,
                variant,
            )
        )


actual_execution_sequence = []


sorted_worker_rows = sorted(
    worker_rows,
    key=lambda row: int(
        row[
            "worker_execution_index"
        ]
    ),
)


for expected_index, row in enumerate(
    sorted_worker_rows,
    start=1,
):

    if int(
        row[
            "worker_execution_index"
        ]
    ) != expected_index:

        raise RuntimeError(
            "Worker execution index sequence changed."
        )


    actual_execution_sequence.append(
        (
            int(
                row[
                    "pair_block_execution_index"
                ]
            ),

            int(
                row[
                    "batch_size"
                ]
            ),

            int(
                row[
                    "pair_rep"
                ]
            ),

            row[
                "variant"
            ],
        )
    )


if actual_execution_sequence != expected_execution_sequence:

    raise RuntimeError(
        "Actual worker sequence does not exactly match frozen execution plan."
    )


print(
    "50/50 execution slots match frozen randomized order: PASS"
)


# =============================================================================
# 7. WORKER STATUS / FINGERPRINT AUDIT
# =============================================================================

banner(
    "STAGE26-8C2 :: WORKER + PAIR AUDIT"
)


status_counts = Counter(
    row[
        "worker_status"
    ]
    for row in worker_rows
)


print(
    "worker status counts:",
    dict(
        status_counts
    )
)


if status_counts != Counter(
    {
        "PASS":
            50
    }
):

    raise RuntimeError(
        "Not all worker executions PASS."
    )


fingerprint_failures = []


for row in pair_rows:

    if row[
        "V2_status"
    ] != "PASS":

        raise RuntimeError(
            "Pair contains non-PASS V2."
        )


    if row[
        "V3_status"
    ] != "PASS":

        raise RuntimeError(
            "Pair contains non-PASS V3."
        )


    if not bool_csv(
        row[
            "pair_complete"
        ]
    ):

        raise RuntimeError(
            "Pair marked incomplete."
        )


    if not bool_csv(
        row[
            "fingerprint_equal"
        ]
    ):

        fingerprint_failures.append(
            (
                row[
                    "batch_size"
                ],
                row[
                    "pair_rep"
                ],
            )
        )


    if (
        row[
            "V2_fingerprint"
        ]
        !=
        row[
            "V3_fingerprint"
        ]
    ):

        fingerprint_failures.append(
            (
                row[
                    "batch_size"
                ],
                row[
                    "pair_rep"
                ],
            )
        )


if fingerprint_failures:

    raise RuntimeError(
        f"V2/V3 fingerprint mismatch: {fingerprint_failures}"
    )


print(
    "25/25 pair fingerprints equal: PASS"
)


# =============================================================================
# 8. RAW OBSERVATION GEOMETRY
# =============================================================================

banner(
    "STAGE26-8C2 :: RAW OBSERVATION GEOMETRY"
)


expected_timed_runs = {
    1:
        200,

    64:
        150,

    256:
        100,

    1024:
        50,

    8192:
        20,
}


raw_count_by_worker = Counter(
    int(
        row[
            "worker_execution_index"
        ]
    )
    for row in raw_rows
)


for row in worker_rows:

    worker_index = int(
        row[
            "worker_execution_index"
        ]
    )

    batch = int(
        row[
            "batch_size"
        ]
    )

    expected_n = expected_timed_runs[
        batch
    ]


    if int(
        row[
            "n"
        ]
    ) != expected_n:

        raise RuntimeError(
            f"Worker {worker_index}: summary n mismatch."
        )


    if raw_count_by_worker[
        worker_index
    ] != expected_n:

        raise RuntimeError(
            f"Worker {worker_index}: raw observation count mismatch."
        )


print(
    "All 50 workers have exact frozen timed-run counts: PASS"
)

print(
    "Total raw observations:",
    len(
        raw_rows
    ),
    "PASS"
)


# =============================================================================
# 9. EXECUTION CONFIG / RESULT FILE HASH AUDIT
# =============================================================================

banner(
    "STAGE26-8C2 :: PER-WORKER FILE HASH AUDIT"
)


config_files = sorted(
    TRANSIENT_CONFIG_DIR.glob(
        "*_config.json"
    )
)

result_files = sorted(
    TRANSIENT_RESULT_DIR.glob(
        "*_result.json"
    )
)

log_files = sorted(
    TRANSIENT_LOG_DIR.glob(
        "*.log"
    )
)


print(
    "configs :",
    len(
        config_files
    )
)

print(
    "results :",
    len(
        result_files
    )
)

print(
    "logs    :",
    len(
        log_files
    )
)


if len(
    config_files
) != 50:

    raise RuntimeError(
        "Expected 50 execution config files."
    )


if len(
    result_files
) != 50:

    raise RuntimeError(
        "Expected 50 worker result files."
    )


if len(
    log_files
) != 50:

    raise RuntimeError(
        "Expected 50 worker log files."
    )


for row in worker_rows:

    block = int(
        row[
            "pair_block_execution_index"
        ]
    )

    batch = int(
        row[
            "batch_size"
        ]
    )

    pair_rep = int(
        row[
            "pair_rep"
        ]
    )

    variant = row[
        "variant"
    ]


    stem = (
        f"block_{block:02d}"
        f"__B{batch}"
        f"__pair_{pair_rep}"
        f"__{variant.lower()}"
    )


    config_path = (
        TRANSIENT_CONFIG_DIR
        /
        (
            stem
            +
            "_config.json"
        )
    )

    result_path = (
        TRANSIENT_RESULT_DIR
        /
        (
            stem
            +
            "_result.json"
        )
    )

    log_path = (
        TRANSIENT_LOG_DIR
        /
        (
            stem
            +
            ".log"
        )
    )


    for path in [
        config_path,
        result_path,
        log_path,
    ]:

        if not path.is_file():

            raise FileNotFoundError(
                path
            )


    actual_config_sha = sha256_file(
        config_path
    )

    actual_result_sha = sha256_file(
        result_path
    )


    if actual_config_sha != row[
        "execution_config_sha256"
    ]:

        raise RuntimeError(
            f"{stem}: config SHA does not match worker execution table."
        )


    if actual_result_sha != row[
        "result_sha256"
    ]:

        raise RuntimeError(
            f"{stem}: result SHA does not match worker execution table."
        )


    worker_result = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


    if worker_result[
        "status"
    ] != "PASS":

        raise RuntimeError(
            f"{stem}: result JSON is not PASS."
        )


print(
    "50/50 execution configs match recorded hashes: PASS"
)

print(
    "50/50 worker result JSONs match recorded hashes: PASS"
)


# =============================================================================
# 10. FROZEN DESCRIPTIVE SUMMARY AUDIT
# =============================================================================

banner(
    "STAGE26-8C2 :: FROZEN DESCRIPTIVE RESULT AUDIT"
)


batch_by_size = {
    int(
        row[
            "batch_size"
        ]
    ):
        row
    for row in batch_rows
}


if set(
    batch_by_size
) != set(
    BATCHES
):

    raise RuntimeError(
        "Batch summary batch set changed."
    )


throughput_directions = {}


for batch in BATCHES:

    row = batch_by_size[
        batch
    ]

    expected = EXPECTED_BATCH_MEDIANS[
        batch
    ]


    if int(
        row[
            "frozen_pair_count"
        ]
    ) != 5:

        raise RuntimeError(
            f"B={batch}: frozen pair count changed."
        )


    if int(
        row[
            "complete_pair_count"
        ]
    ) != 5:

        raise RuntimeError(
            f"B={batch}: complete pair count changed."
        )


    if int(
        row[
            "incomplete_pair_count"
        ]
    ) != 0:

        raise RuntimeError(
            f"B={batch}: incomplete pair unexpectedly present."
        )


    if not close_float(
        row[
            "throughput_ratio_median"
        ],
        expected[
            "throughput"
        ],
    ):

        raise RuntimeError(
            f"B={batch}: throughput median mismatch."
        )


    if not close_float(
        row[
            "p50_ratio_median"
        ],
        expected[
            "p50"
        ],
    ):

        raise RuntimeError(
            f"B={batch}: p50 median mismatch."
        )


    if not close_float(
        row[
            "p95_ratio_median"
        ],
        expected[
            "p95"
        ],
    ):

        raise RuntimeError(
            f"B={batch}: p95 median mismatch."
        )


    expected_p99 = expected[
        "p99"
    ]


    if expected_p99 is None:

        if row[
            "p99_ratio_median"
        ] not in (
            "",
            None,
        ):

            raise RuntimeError(
                f"B={batch}: p99 should be N/A."
            )

    else:

        if not close_float(
            row[
                "p99_ratio_median"
            ],
            expected_p99,
        ):

            raise RuntimeError(
                f"B={batch}: p99 median mismatch."
            )


    throughput = float(
        row[
            "throughput_ratio_median"
        ]
    )


    if throughput > 1:

        direction = (
            "V3_MEDIAN_THROUGHPUT_GREATER_THAN_V2"
        )

    elif throughput < 1:

        direction = (
            "V3_MEDIAN_THROUGHPUT_LESS_THAN_V2"
        )

    else:

        direction = (
            "V3_MEDIAN_THROUGHPUT_EQUAL_TO_V2"
        )


    throughput_directions[
        str(
            batch
        )
    ] = direction


    print(
        f"B={batch:5d} "
        f"throughput={throughput:.9f} "
        f"p50={float(row['p50_ratio_median']):.9f} "
        f"p95={float(row['p95_ratio_median']):.9f} "
        f"direction={direction}"
    )


if len(
    set(
        throughput_directions.values()
    )
) <= 1:

    raise RuntimeError(
        "Expected observed sensitivity direction to differ across batches."
    )


print(
    "\nSensitivity direction across batches:"
)

print(
    "  B=1 and B=8192 : throughput ratio > 1"
)

print(
    "  B=64/256/1024 : throughput ratio < 1"
)

print(
    "  directionally uniform across batches: NO"
)


# =============================================================================
# 11. STATISTICAL / SCIENTIFIC BOUNDARY AUDIT
# =============================================================================

banner(
    "STAGE26-8C2 :: SCIENTIFIC BOUNDARY AUDIT"
)


scientific_checks = {
    "protocol bootstrap false":
        (
            protocol[
                "inference_policy"
            ][
                "bootstrap"
            ]
            is False
        ),

    "protocol hypothesis test false":
        (
            protocol[
                "inference_policy"
            ][
                "hypothesis_test"
            ]
            is False
        ),

    "protocol materiality threshold none":
        (
            protocol[
                "inference_policy"
            ][
                "materiality_threshold"
            ]
            is None
        ),

    "execution bootstrap false":
        (
            execution_receipt[
                "scientific_state"
            ][
                "bootstrap_executed"
            ]
            is False
        ),

    "execution hypothesis false":
        (
            execution_receipt[
                "scientific_state"
            ][
                "hypothesis_test_performed"
            ]
            is False
        ),

    "execution materiality threshold false":
        (
            execution_receipt[
                "scientific_state"
            ][
                "materiality_threshold_selected"
            ]
            is False
        ),

    "historical 4C3 unmodified":
        (
            execution_receipt[
                "scientific_state"
            ][
                "historical_4C3_modified"
            ]
            is False
        ),

    "historical 4C3 not replaced":
        (
            execution_receipt[
                "scientific_state"
            ][
                "historical_4C3_replaced"
            ]
            is False
        ),

    "Stage26-7C not recomputed":
        (
            execution_receipt[
                "scientific_state"
            ][
                "Stage26_7C_recomputed"
            ]
            is False
        ),

    "model not loaded":
        (
            execution_receipt[
                "scientific_state"
            ][
                "model_loaded"
            ]
            is False
        ),

    "inference false":
        (
            execution_receipt[
                "scientific_state"
            ][
                "inference_performed"
            ]
            is False
        ),

    "Pareto false":
        (
            execution_receipt[
                "scientific_state"
            ][
                "Pareto_recomputed"
            ]
            is False
        ),

    "PCAP false":
        (
            execution_receipt[
                "scientific_state"
            ][
                "PCAP_accessed"
            ]
            is False
        ),

    "labels false":
        (
            execution_receipt[
                "scientific_state"
            ][
                "labels_accessed"
            ]
            is False
        ),

    "GPU false":
        (
            execution_receipt[
                "scientific_state"
            ][
                "GPU_used"
            ]
            is False
        ),
}


for label, passed in scientific_checks.items():

    print(
        f"{label:42s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    scientific_checks.values()
):

    raise RuntimeError(
        "Stage26-8C1 scientific boundary audit failed."
    )


# =============================================================================
# 12. CREATE DURABLE CHECKPOINT
# =============================================================================

banner(
    "STAGE26-8C2 :: COPY EXACT RAW EVIDENCE"
)


CHECKPOINT.mkdir(
    parents=True,
    exist_ok=False,
)

DURABLE_EVIDENCE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

DURABLE_PROVENANCE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# Copy exact top-level measurement products.
for source in [
    WORKER_EXECUTION_CSV,
    RAW_OBSERVATIONS_CSV,
    PAIR_RESULTS_CSV,
    BATCH_SUMMARY_CSV,
    RESULTS_JSON,
    EXECUTION_RECEIPT,
]:

    destination = (
        DURABLE_EVIDENCE_DIR
        /
        source.name
    )

    shutil.copy2(
        source,
        destination,
    )


    if sha256_file(
        destination
    ) != sha256_file(
        source
    ):

        raise RuntimeError(
            f"Byte copy mismatch: {source.name}"
        )


# Copy all execution-specific raw evidence.
shutil.copytree(
    TRANSIENT_CONFIG_DIR,
    DURABLE_CONFIG_DIR,
)

shutil.copytree(
    TRANSIENT_RESULT_DIR,
    DURABLE_RESULT_DIR,
)

shutil.copytree(
    TRANSIENT_LOG_DIR,
    DURABLE_LOG_DIR,
)


# Copy exact input-restoration receipt.
shutil.copy2(
    RESTORE_RECEIPT,
    DURABLE_RESTORE_RECEIPT,
)


if sha256_file(
    DURABLE_RESTORE_RECEIPT
) != EXPECTED_RESTORE_RECEIPT_SHA256:

    raise RuntimeError(
        "Durable restoration receipt copy mismatch."
    )


# =============================================================================
# 13. DESCRIPTIVE SENSITIVITY AUDIT RECORD
# =============================================================================

banner(
    "STAGE26-8C2 :: WRITE DESCRIPTIVE AUDIT"
)


audit_payload = {
    "schema":
        "stage26_8c1_paired_representation_sensitivity_audit_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-8C2",

    "scientific_parent":
        EXPECTED_PARENT,

    "status":
        "PASS_PAIRED_4C3_IMPLEMENTATION_SENSITIVITY_AUDIT_COMPLETED",

    "trigger": {
        "historical_issue":
            "OUTSIDE_TIMER_BEFORE_NEXT_TIMED_ITERATION",

        "historical_worker":
            "V2_ORIGINAL",

        "historical_worker_sha256":
            EXPECTED_V2_SHA256,

        "sensitivity_worker":
            "V3_CLEAN",

        "sensitivity_worker_sha256":
            EXPECTED_V3_SHA256,
    },

    "execution_geometry": {
        "frozen_pair_blocks":
            25,

        "worker_execution_slots":
            50,

        "worker_PASS":
            50,

        "complete_pairs":
            25,

        "incomplete_pairs":
            0,

        "replacement_pairs":
            0,

        "all_pair_fingerprints_equal":
            True,
    },

    "descriptive_batch_results": {
        str(
            batch
        ): {
            "throughput_ratio_V3_over_V2_median":
                float(
                    batch_by_size[
                        batch
                    ][
                        "throughput_ratio_median"
                    ]
                ),

            "throughput_ratio_V3_over_V2_min":
                float(
                    batch_by_size[
                        batch
                    ][
                        "throughput_ratio_min"
                    ]
                ),

            "throughput_ratio_V3_over_V2_max":
                float(
                    batch_by_size[
                        batch
                    ][
                        "throughput_ratio_max"
                    ]
                ),

            "p50_latency_ratio_V3_over_V2_median":
                float(
                    batch_by_size[
                        batch
                    ][
                        "p50_ratio_median"
                    ]
                ),

            "p95_latency_ratio_V3_over_V2_median":
                float(
                    batch_by_size[
                        batch
                    ][
                        "p95_ratio_median"
                    ]
                ),

            "p99_latency_ratio_V3_over_V2_median":
                (
                    None
                    if
                    batch_by_size[
                        batch
                    ][
                        "p99_ratio_median"
                    ]
                    in
                    (
                        "",
                        None,
                    )
                    else
                    float(
                        batch_by_size[
                            batch
                        ][
                            "p99_ratio_median"
                        ]
                    )
                ),

            "throughput_direction":
                throughput_directions[
                    str(
                        batch
                    )
                ],
        }
        for batch in BATCHES
    },

    "descriptive_interpretation": {
        "throughput_direction_uniform_across_batches":
            False,

        "supported_statement":
            (
                "Removing the historical inter-iteration full-output "
                "fingerprint changes measured representation timing in a "
                "batch-dependent direction in this contemporaneous paired "
                "sensitivity experiment."
            ),

        "unsupported_statements": [
            "V3_IS_STATISTICALLY_FASTER",
            "V3_IS_STATISTICALLY_SLOWER",
            "HISTORICAL_4C3_IS_BIASED_BY_A_FIXED_PERCENTAGE",
            "V3_IS_A_CORRECTED_REPLACEMENT_FOR_HISTORICAL_4C3",
            "SENSITIVITY_IS_MATERIALLY_NEGLIGIBLE",
            "SENSITIVITY_IS_MATERIALLY_LARGE",
        ],
    },

    "statistical_policy": {
        "descriptive_only":
            True,

        "bootstrap":
            False,

        "confidence_interval":
            False,

        "hypothesis_test":
            False,

        "p_value":
            False,

        "materiality_threshold":
            None,

        "post_hoc_threshold":
            False,
    },

    "publication_policy": {
        "historical_stage26_4c3":
            "RETAIN_AS_ORIGINAL_MEASUREMENT",

        "V3":
            "PAIRED_SENSITIVITY_ONLY_NOT_REPLACEMENT",

        "stage26_7c":
            "UNCHANGED",

        "stage26_7_group_b_component_ratios":
            "UNCHANGED_HISTORICAL_4C3_BASED_COMPONENT_RESULTS",

        "stage26_8_4c3_audit_item":
            "COMPLETED_WITH_PAIRED_SENSITIVITY_EVIDENCE",

        "publication_label_after_CPU_closure":
            (
                "COMPONENT_LEVEL_WITH_STAGE26_8_4C3_"
                "SENSITIVITY_AUDIT_COMPLETED"
            ),

        "complete_E2E_claim_allowed":
            False,
    },

    "scientific_state": {
        "new_timing_in_this_anchor_cell":
            False,

        "sensitivity_timing_already_completed":
            True,

        "historical_4C3_modified":
            False,

        "historical_4C3_replaced":
            False,

        "Stage26_7C_recomputed":
            False,

        "bootstrap_executed":
            False,

        "hypothesis_test_performed":
            False,

        "materiality_threshold_selected":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "Pareto_recomputed":
            False,

        "PCAP_accessed":
            False,

        "labels_accessed":
            False,

        "GPU_used":
            False,
    },
}


atomic_json(
    AUDIT_JSON,
    audit_payload,
)


audit_sha = sha256_file(
    AUDIT_JSON
)


print(
    "Audit SHA256:",
    audit_sha
)


# =============================================================================
# 14. ANCHOR RECEIPT
# =============================================================================

anchor_receipt_payload = {
    "schema":
        "stage26_8c2_paired_sensitivity_anchor_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-8C2",

    "status":
        "READY_FOR_GIT_ANCHOR",

    "scientific_parent":
        EXPECTED_PARENT,

    "commit_subject":
        COMMIT_SUBJECT,

    "frozen_upstream": {
        "protocol_sha256":
            EXPECTED_PROTOCOL_SHA256,

        "execution_plan_sha256":
            EXPECTED_EXECUTION_PLAN_SHA256,

        "freeze_receipt_sha256":
            EXPECTED_FREEZE_RECEIPT_SHA256,

        "lock_manifest_sha256":
            EXPECTED_LOCK_MANIFEST_SHA256,

        "V2_worker_sha256":
            EXPECTED_V2_SHA256,

        "V3_worker_sha256":
            EXPECTED_V3_SHA256,

        "restore_receipt_sha256":
            EXPECTED_RESTORE_RECEIPT_SHA256,
    },

    "completed_measurement_top_level_hashes":
        EXPECTED_TOP_LEVEL_HASHES,

    "execution_geometry": {
        "worker_executions":
            50,

        "worker_PASS":
            50,

        "raw_observations":
            EXPECTED_RAW_OBSERVATIONS,

        "pair_blocks":
            25,

        "complete_pairs":
            25,

        "incomplete_pairs":
            0,

        "replacement_pairs":
            0,

        "all_pair_fingerprints_equal":
            True,
    },

    "sensitivity_audit_sha256":
        audit_sha,

    "scientific_state": {
        "new_timing_executed_by_anchor":
            False,

        "completed_timing_rerun":
            False,

        "pair_replacement":
            False,

        "historical_4C3_modified":
            False,

        "historical_4C3_replaced":
            False,

        "Stage26_7C_recomputed":
            False,

        "bootstrap_executed":
            False,

        "hypothesis_test_performed":
            False,

        "materiality_threshold_selected":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "Pareto_recomputed":
            False,

        "PCAP_accessed":
            False,

        "labels_accessed":
            False,

        "GPU_used":
            False,
    },

    "next":
        (
            "After remote byte/scientific verification, proceed to final "
            "Stage26 CPU closure audit and publication-facing CPU tables/"
            "figures. Do not modify historical Stage26-4C3 or Stage26-7C."
        ),
}


atomic_json(
    ANCHOR_RECEIPT,
    anchor_receipt_payload,
)


anchor_receipt_sha = sha256_file(
    ANCHOR_RECEIPT
)


# =============================================================================
# 15. BUILD COMPLETE MANIFEST
# =============================================================================

banner(
    "STAGE26-8C2 :: BUILD DURABLE MANIFEST"
)


manifest_files = []


for path in sorted(
    p
    for p in CHECKPOINT.rglob(
        "*"
    )
    if p.is_file()
    and
    p != MANIFEST
):

    manifest_files.append(
        {
            "repo_relative_path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


print(
    "Manifest file count excluding manifest:",
    len(
        manifest_files
    )
)


# Expected:
#   6 top-level evidence
# + 50 configs
# + 50 results
# + 50 logs
# + 1 restore receipt
# + 1 audit
# + 1 anchor receipt
# = 159
EXPECTED_MANIFEST_FILE_COUNT = (
    159
)


if len(
    manifest_files
) != EXPECTED_MANIFEST_FILE_COUNT:

    raise RuntimeError(
        f"Expected {EXPECTED_MANIFEST_FILE_COUNT} manifest files; "
        f"found {len(manifest_files)}."
    )


manifest_payload = {
    "schema":
        "stage26_8c1_paired_representation_sensitivity_manifest_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage":
        26,

    "checkpoint":
        "STAGE26-8C2",

    "status":
        "READY_FOR_GIT_ANCHOR",

    "scientific_parent":
        EXPECTED_PARENT,

    "commit_subject":
        COMMIT_SUBJECT,

    "file_count_excluding_manifest":
        len(
            manifest_files
        ),

    "audit_sha256":
        audit_sha,

    "anchor_receipt_sha256":
        anchor_receipt_sha,

    "files":
        manifest_files,

    "scientific_state": {
        "raw_evidence_copied_byte_identically":
            True,

        "sensitivity_results_modified":
            False,

        "timing_rerun":
            False,

        "historical_4C3_modified":
            False,

        "Stage26_7C_modified":
            False,

        "GPU_used":
            False,
    },
}


atomic_json(
    MANIFEST,
    manifest_payload,
)


manifest_sha = sha256_file(
    MANIFEST
)


print(
    "Anchor receipt SHA256:",
    anchor_receipt_sha
)

print(
    "Manifest SHA256      :",
    manifest_sha
)


# =============================================================================
# 16. LOCAL BYTE AUDIT
# =============================================================================

banner(
    "STAGE26-8C2 :: LOCAL DURABLE BYTE AUDIT"
)


for row in manifest_files:

    path = (
        REPO
        /
        row[
            "repo_relative_path"
        ]
    )


    actual_size = int(
        path.stat().st_size
    )

    actual_sha = sha256_file(
        path
    )


    passed = (
        actual_size
        ==
        int(
            row[
                "size_bytes"
            ]
        )
        and
        actual_sha
        ==
        row[
            "sha256"
        ]
    )


    if not passed:

        raise RuntimeError(
            f"Local manifest verification failed: {path}"
        )


print(
    f"{len(manifest_files)}/{len(manifest_files)} "
    "manifest-listed files byte-verified: PASS"
)


# Explicit top-level source-to-durable identity check.
for filename, expected_sha in EXPECTED_TOP_LEVEL_HASHES.items():

    durable = (
        DURABLE_EVIDENCE_DIR
        /
        filename
    )


    if sha256_file(
        durable
    ) != expected_sha:

        raise RuntimeError(
            f"Durable top-level evidence changed: {filename}"
        )


print(
    "6/6 completed Stage26-8C1 top-level artifacts copied unchanged: PASS"
)


# =============================================================================
# 17. GIT CHANGE AUDIT
# =============================================================================

banner(
    "STAGE26-8C2 :: GIT CHANGE AUDIT"
)


repo_status = git(
    "status",
    "--porcelain",
)


print(
    repo_status
)


if not repo_status:

    raise RuntimeError(
        "Expected uncommitted Stage26-8C2 checkpoint."
    )


unexpected = []


for line in repo_status.splitlines():

    relpath = line[
        3:
    ]


    if not relpath.startswith(
        str(
            CHECKPOINT_REL
        )
        +
        "/"
    ):

        unexpected.append(
            line
        )


if unexpected:

    raise RuntimeError(
        "Unexpected repository changes:\n"
        +
        "\n".join(
            unexpected
        )
    )


# =============================================================================
# 18. GIT IDENTITY
# =============================================================================

author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_PARENT,
)

author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_PARENT,
)


if (
    not author_name.strip()
    or
    "@"
    not in
    author_email
):

    raise RuntimeError(
        "Could not recover Git author identity."
    )


git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


print(
    "Git author:",
    author_name,
    "<" + author_email + ">"
)


# =============================================================================
# 19. COMMIT
# =============================================================================

banner(
    "STAGE26-8C2 :: COMMIT"
)


git(
    "add",
    str(
        CHECKPOINT_REL
    ),
)


staged = git(
    "diff",
    "--cached",
    "--name-status",
)


print(
    staged
)


git(
    "commit",
    "-m",
    COMMIT_SUBJECT,
)


commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    "HEAD^",
)

commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "\nCommit :",
    commit_sha
)

print(
    "Parent :",
    commit_parent
)

print(
    "Subject:",
    commit_subject
)


if commit_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-8C2 commit parent mismatch."
    )


if commit_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Stage26-8C2 commit subject mismatch."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository not clean after Stage26-8C2 commit."
    )


# =============================================================================
# 20. PUSH USING KAGGLE SECRET
# =============================================================================

banner(
    "STAGE26-8C2 :: PUSH"
)


from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not github_token:

    raise RuntimeError(
        "Kaggle GITHUB_TOKEN unavailable."
    )


print(
    "GITHUB_TOKEN: FOUND"
)

print(
    "Token value : <not printed>"
)


with tempfile.TemporaryDirectory(
    prefix="stage26_git_auth_"
) as tmpdir:

    tmpdir = Path(
        tmpdir
    )

    askpass = (
        tmpdir
        /
        "askpass.sh"
    )


    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *sername*) printf '%s\\n' "x-access-token" ;;
  *assword*) printf '%s\\n' "$STAGE26_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
    )


    push_env = os.environ.copy()

    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token


    push_result = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
    )


    print(
        push_result.stdout.strip()
    )


github_token = None


# =============================================================================
# 21. REMOTE COMMIT VERIFICATION
# =============================================================================

banner(
    "STAGE26-8C2 :: REMOTE COMMIT VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

remote_parent = git(
    "rev-parse",
    "origin/main^",
)

remote_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)

print(
    "Parent     :",
    remote_parent
)

print(
    "Subject    :",
    remote_subject
)


if local_after != commit_sha:

    raise RuntimeError(
        "Local HEAD changed after push."
    )


if remote_after != commit_sha:

    raise RuntimeError(
        "origin/main did not advance to Stage26-8C2."
    )


if remote_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Remote Stage26-8C2 parent mismatch."
    )


if remote_subject != COMMIT_SUBJECT:

    raise RuntimeError(
        "Remote Stage26-8C2 subject mismatch."
    )


# =============================================================================
# 22. REMOTE MANIFEST + BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-8C2 :: REMOTE BYTE VERIFICATION"
)


remote_manifest_bytes = git_blob_bytes(
    "origin/main",
    str(
        MANIFEST.relative_to(
            REPO
        )
    ),
)


remote_manifest_sha = sha256_bytes(
    remote_manifest_bytes
)


print(
    "Expected manifest SHA256:",
    manifest_sha
)

print(
    "Remote manifest SHA256  :",
    remote_manifest_sha
)


if remote_manifest_sha != manifest_sha:

    raise RuntimeError(
        "Remote manifest SHA mismatch."
    )


remote_manifest = json.loads(
    remote_manifest_bytes.decode(
        "utf-8"
    )
)


remote_pass = 0


for row in remote_manifest[
    "files"
]:

    data = git_blob_bytes(
        "origin/main",
        row[
            "repo_relative_path"
        ],
    )


    actual_size = len(
        data
    )

    actual_sha = sha256_bytes(
        data
    )


    if (
        actual_size
        !=
        int(
            row[
                "size_bytes"
            ]
        )
        or
        actual_sha
        !=
        row[
            "sha256"
        ]
    ):

        raise RuntimeError(
            "Remote byte verification failed:\n"
            +
            row[
                "repo_relative_path"
            ]
        )


    remote_pass += 1


print(
    f"Remote manifest files verified: "
    f"{remote_pass}/{len(remote_manifest['files'])} PASS"
)


# =============================================================================
# 23. REMOTE SCIENTIFIC VERIFICATION
# =============================================================================

banner(
    "STAGE26-8C2 :: REMOTE SCIENTIFIC VERIFICATION"
)


remote_audit = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            AUDIT_JSON.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)

remote_anchor = json.loads(
    git_blob_bytes(
        "origin/main",
        str(
            ANCHOR_RECEIPT.relative_to(
                REPO
            )
        ),
    ).decode(
        "utf-8"
    )
)


remote_checks = {
    "50 worker PASS":
        (
            remote_audit[
                "execution_geometry"
            ][
                "worker_PASS"
            ]
            ==
            50
        ),

    "25 complete pairs":
        (
            remote_audit[
                "execution_geometry"
            ][
                "complete_pairs"
            ]
            ==
            25
        ),

    "zero incomplete pairs":
        (
            remote_audit[
                "execution_geometry"
            ][
                "incomplete_pairs"
            ]
            ==
            0
        ),

    "zero replacement pairs":
        (
            remote_audit[
                "execution_geometry"
            ][
                "replacement_pairs"
            ]
            ==
            0
        ),

    "all fingerprints equal":
        (
            remote_audit[
                "execution_geometry"
            ][
                "all_pair_fingerprints_equal"
            ]
            is True
        ),

    "batch-dependent direction recorded":
        (
            remote_audit[
                "descriptive_interpretation"
            ][
                "throughput_direction_uniform_across_batches"
            ]
            is False
        ),

    "historical 4C3 retained":
        (
            remote_audit[
                "publication_policy"
            ][
                "historical_stage26_4c3"
            ]
            ==
            "RETAIN_AS_ORIGINAL_MEASUREMENT"
        ),

    "V3 not replacement":
        (
            remote_audit[
                "publication_policy"
            ][
                "V3"
            ]
            ==
            "PAIRED_SENSITIVITY_ONLY_NOT_REPLACEMENT"
        ),

    "Stage26-7C unchanged":
        (
            remote_audit[
                "publication_policy"
            ][
                "stage26_7c"
            ]
            ==
            "UNCHANGED"
        ),

    "4C3 audit completed":
        (
            remote_audit[
                "publication_policy"
            ][
                "stage26_8_4c3_audit_item"
            ]
            ==
            "COMPLETED_WITH_PAIRED_SENSITIVITY_EVIDENCE"
        ),

    "no bootstrap":
        (
            remote_audit[
                "statistical_policy"
            ][
                "bootstrap"
            ]
            is False
        ),

    "no hypothesis test":
        (
            remote_audit[
                "statistical_policy"
            ][
                "hypothesis_test"
            ]
            is False
        ),

    "no materiality threshold":
        (
            remote_audit[
                "statistical_policy"
            ][
                "materiality_threshold"
            ]
            is None
        ),

    "no timing rerun in anchor":
        (
            remote_anchor[
                "scientific_state"
            ][
                "completed_timing_rerun"
            ]
            is False
        ),

    "GPU false":
        (
            remote_anchor[
                "scientific_state"
            ][
                "GPU_used"
            ]
            is False
        ),
}


for label, passed in remote_checks.items():

    print(
        f"{label:40s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    remote_checks.values()
):

    raise RuntimeError(
        "Remote Stage26-8C2 scientific verification failed."
    )


# =============================================================================
# 24. FINAL CLOSURE
# =============================================================================

banner(
    "STAGE26-8C2 PAIRED SENSITIVITY ANCHOR COMPLETE"
)


final_status = git(
    "status",
    "--porcelain",
)


print(
    "NEW DURABLE COMMIT:"
)

print(
    " ",
    commit_sha
)


print(
    "\nPARENT:"
)

print(
    " ",
    EXPECTED_PARENT
)


print(
    "\nEXECUTION EVIDENCE:"
)

print(
    "  workers                  : 50 / 50 PASS"
)

print(
    "  frozen pair blocks       : 25 / 25 complete"
)

print(
    "  incomplete pairs         : 0"
)

print(
    "  replacement pairs        : 0"
)

print(
    "  raw observations         :",
    EXPECTED_RAW_OBSERVATIONS
)

print(
    "  V2/V3 fingerprints equal : 25 / 25"
)


print(
    "\nPAIRED THROUGHPUT MEDIANS V3/V2:"
)

for batch in BATCHES:

    value = float(
        batch_by_size[
            batch
        ][
            "throughput_ratio_median"
        ]
    )


    print(
        f"  B={batch:5d} : {value:.9f}"
    )


print(
    "\nSCIENTIFIC INTERPRETATION:"
)

print(
    "  effect direction uniform across batches : NO"
)

print(
    "  historical Stage26-4C3                  : RETAINED"
)

print(
    "  V3                                      : SENSITIVITY ONLY"
)

print(
    "  Stage26-7C                              : UNCHANGED"
)

print(
    "  4C3 sensitivity audit                   : COMPLETED"
)

print(
    "  full E2E claim                          : NO"
)


print(
    "\nSTATISTICS:"
)

print(
    "  bootstrap            : NO"
)

print(
    "  hypothesis test      : NO"
)

print(
    "  p-value              : NO"
)

print(
    "  materiality threshold: NONE"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  timing rerun by anchor   : NO"
)

print(
    "  historical 4C3 modified  : NO"
)

print(
    "  historical 4C3 replaced  : NO"
)

print(
    "  Stage26-7C recomputed    : NO"
)

print(
    "  inference                : NO"
)

print(
    "  Pareto                   : NO"
)

print(
    "  PCAP                     : NO"
)

print(
    "  labels                   : NO"
)

print(
    "  GPU                      : NO"
)


print(
    "\nHASHES:"
)

print(
    "  audit         :",
    audit_sha
)

print(
    "  anchor receipt:",
    anchor_receipt_sha
)

print(
    "  manifest      :",
    manifest_sha
)


print(
    "\nREMOTE VERIFICATION:"
)

print(
    "  commit          : PASS"
)

print(
    "  parent          : PASS"
)

print(
    "  manifest        : PASS"
)

print(
    "  all raw evidence: PASS"
)

print(
    "  science policy  : PASS"
)


print(
    "\nRepo clean:",
    final_status == ""
)


if final_status:

    raise RuntimeError(
        "Repository not clean after Stage26-8C2."
    )


print(
    "\nNEXT:"
)

print(
    "  Stage26-4C3 implementation-sensitivity audit is now complete."
)

print(
    "  Proceed to Stage26-8 final CPU closure:"
)

print(
    "    - complete hash/protocol audit"
)

print(
    "    - publication-facing CPU tables"
)

print(
    "    - publication-facing CPU figures"
)

print(
    "    - resource-limit outcomes retained"
)

print(
    "    - corrected Stage26-2 uncertainty only"
)

print(
    "    - no complete-E2E claim"
)

print(
    "    - final CPU closure commit/push"
)

print(
    "  GPU remains OFF until that closure is remotely verified."
)


STAGE26-8C2 :: DURABLE SCIENTIFIC PARENT
Expected parent: 973e0c479ce91ad21e92ed5db11585f26a4c49ec
Local HEAD     : 973e0c479ce91ad21e92ed5db11585f26a4c49ec
origin/main    : 973e0c479ce91ad21e92ed5db11585f26a4c49ec
Repo clean     : True

STAGE26-8C2 :: UPSTREAM IDENTITY
protocol           PASS 4ce96c1540b29bd9683305ed981412b46be4b0b2596dcb58b3064f0724f05c5e
execution plan     PASS 5581bb2576919930913f49a308470083e7b27e315bc5cd99ba1fc1a853dcac14
freeze receipt     PASS d052bcd8bf3b2ed6a93f1ca3813036c030fa1d1de1fb28ffea166cf3e226bc33
lock manifest      PASS 6cfa4790ccabd1d755af40960b2e38846b4e15fe92ba2fac70d72ac3c162334f
V2 worker          PASS ebc19f2ea82b8cbc1b26c9cf6797550293eb16d16ae41e649cbd8d154d9ccd23
V3 worker          PASS c065e70fc16a628e482961f0db669d95affb841edd39153abf28dc99e868ad02
restore receipt    PASS babcf2331db7c242297b346bad0904bb061251acb83003b5222c16295de96ff1

STAGE26-8C2 :: EXACT COMPLETED-MEASUREMENT HASH GATE
stage26_8c1_worker_executions.csv                  

In [12]:
# =============================================================================
# STAGE26-8D0
# FINAL CPU PUBLICATION-SOURCE / UNCERTAINTY / BOUNDARY AUDIT
#
# DURABLE SCIENTIFIC PARENT:
#   fd4497e333ebeb72b1eb188ed64ca2cdd0f08567
#
# PURPOSE
# -------
# Before generating ANY publication-facing CPU table or figure:
#
#   1. verify the new Stage26-8C2 durable anchor;
#   2. inventory every Stage26 result family relevant to final CPU reporting;
#   3. verify the completed 4C3 sensitivity closure;
#   4. identify the exact corrected Stage26-2 uncertainty package;
#   5. audit Stage26-1 cold-start CI/bootstrap provenance separately;
#   6. freeze what may / may not enter publication-facing CPU outputs;
#   7. preserve OOM / timeout as resource-limit outcomes;
#   8. preserve Stage26-5A complete-E2E closure;
#   9. preserve immutable Stage26-6D Pareto;
#  10. preserve Stage26-7C component-only capacity/scaling interpretation.
#
# THIS CELL IS READ ONLY WITH RESPECT TO GIT.
#
# It writes ONE TRANSIENT audit outside the repository:
#
#   /kaggle/working/stage26_deployment_profiling/cpu_closure/
#       stage26_8d0_cpu_publication_source_audit.json
#
# NO:
#   - timing
#   - inference
#   - model loading
#   - bootstrap
#   - new CI calculation
#   - new ratio calculation
#   - Pareto recomputation
#   - publication table generation
#   - publication figure generation
#   - PCAP
#   - labels
#   - GPU
#   - Git modification
# =============================================================================

from __future__ import annotations

import hashlib
import json
import os
import re
import subprocess
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

RESULT_ROOT = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
)

EXPECTED_PARENT = (
    "fd4497e333ebeb72b1eb188ed64ca2cdd0f08567"
)

RUNTIME_AUDIT = Path(
    "/kaggle/working/stage26_deployment_profiling/"
    "cpu_closure/"
    "stage26_8d0_cpu_publication_source_audit.json"
)


# -----------------------------------------------------------------------------
# Exact Stage26-8C2 identities
# -----------------------------------------------------------------------------

SENS_DIR = (
    RESULT_ROOT
    / "stage26_8c1_paired_representation_sensitivity"
)

SENS_AUDIT = (
    SENS_DIR
    / "stage26_8c1_sensitivity_audit.json"
)

SENS_ANCHOR_RECEIPT = (
    SENS_DIR
    / "stage26_8c2_anchor_receipt.json"
)

SENS_MANIFEST = (
    SENS_DIR
    / "stage26_8c1_paired_sensitivity_manifest.json"
)


EXPECTED_SENS_AUDIT_SHA256 = (
    "7daaa517e49ca99787b124855bb425abc89b0ef4522556a7500c69f9fd962af2"
)

EXPECTED_SENS_ANCHOR_RECEIPT_SHA256 = (
    "45d6d5e5fbb1dfccc08dc64449b9ec1af1951799208770824cc1cc702991e8ad"
)

EXPECTED_SENS_MANIFEST_SHA256 = (
    "82af40d79f713013e9914b62aa6d0683944bc19ec2310050d2733be2ee244982"
)


# -----------------------------------------------------------------------------
# Exact Stage26-7C identities
# -----------------------------------------------------------------------------

CAPACITY_DIR = (
    RESULT_ROOT
    / "stage26_7c_cpu_capacity_scaling"
)

CAPACITY_RESULTS = (
    CAPACITY_DIR
    / "stage26_7c_capacity_scaling_results.json"
)

CAPACITY_RECEIPT = (
    CAPACITY_DIR
    / "stage26_7c_capacity_scaling_receipt.json"
)

CAPACITY_MANIFEST = (
    CAPACITY_DIR
    / "stage26_7c_capacity_scaling_manifest.json"
)


EXPECTED_CAPACITY_RESULTS_SHA256 = (
    "ee5a852fb43ce01c980e64ef83611a88808d3a11a56a078609467fb899e39fdf"
)

EXPECTED_CAPACITY_RECEIPT_SHA256 = (
    "30bfd4657a5a0218ce22a4fab1922ee26e07496c148809ec04c8059656b73ce4"
)

EXPECTED_CAPACITY_MANIFEST_SHA256 = (
    "f090a992195e718b9ae569e681b3fd4b50b9dba76bca89fb6fc1099e30fa426c"
)


# -----------------------------------------------------------------------------
# Exact Stage26-6F1 corrected timing uncertainty identities
# -----------------------------------------------------------------------------
#
# We deliberately resolve these BY SHA rather than assuming a directory name.
# The required known hashes are frozen from the completed/pushed 6F1 package.
# -----------------------------------------------------------------------------

EXPECTED_6F1_HASHES = {
    "corrected_results_json":
        "45ea4408738fb5101ce68a363c2cc0a7951648eed49e474da9489b00ce6128ce",

    "condition_csv":
        "f804f117312b967bfc13cc070a34be43545238bfd30a758a00ad58202c9e6828",

    "six_point_csv":
        "1d3870b62187037757ee3c98d126c2a004edb01c858b2ec22c2e54e7b4e52937",

    "receipt":
        "8d37b6bc2e0a7a75b8102b7d1cd3c1c058907420b229c9da1f302d97a44f1130",

    "manifest":
        "94f1acd338c933dbc7d45233c8f6a8429e37df1cdc2a9cca356b730a00c573e2",
}


# -----------------------------------------------------------------------------
# Exact Stage26-4C3 historical identities
# -----------------------------------------------------------------------------

REP_4C3_DIR = (
    RESULT_ROOT
    / "stage26_4c3_cpu1_representation_timing"
)

REP_4C3_SUMMARY_CSV = (
    REP_4C3_DIR
    / "stage26_4c3_cpu1_representation_summary.csv"
)

REP_4C3_SUMMARY_JSON = (
    REP_4C3_DIR
    / "stage26_4c3_cpu1_representation_summary.json"
)


EXPECTED_REP_4C3_SUMMARY_CSV_SHA256 = (
    "ca5bbc17edfa4bb088599236b4099c1986fd73cb7f087ef3ac753a524afd2ba5"
)

EXPECTED_REP_4C3_SUMMARY_JSON_SHA256 = (
    "63c8041144668c62218fc9c46eb6a57aba9dcda0c9a3f0477f932f6a572c34f7"
)


# -----------------------------------------------------------------------------
# Exact raw-extraction Stage26-4B3 summary identity
# -----------------------------------------------------------------------------

EXPECTED_RAW_EXTRACTION_SUMMARY_SHA256 = (
    "1914504fad850879d1ed11afed432436c18454f2bb8650f881244c865f18ba16"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def git(*args):

    p = subprocess.run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def atomic_json(path, payload):

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path)
        +
        ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def flatten_json(
    obj,
    *,
    prefix="$",
    result=None,
):

    if result is None:

        result = []


    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            flatten_json(
                value,
                prefix=(
                    prefix
                    +
                    "."
                    +
                    str(
                        key
                    )
                ),
                result=result,
            )


    elif isinstance(
        obj,
        list,
    ):

        for index, value in enumerate(
            obj
        ):

            flatten_json(
                value,
                prefix=(
                    f"{prefix}[{index}]"
                ),
                result=result,
            )


    else:

        result.append(
            {
                "path":
                    prefix,

                "value":
                    obj,
            }
        )


    return result


def safe_json(path):

    try:

        return json.loads(
            Path(path).read_text(
                encoding="utf-8"
            )
        )

    except Exception:

        return None


def file_inventory(
    root,
    *,
    max_hash_size=20 * 1024 * 1024,
):

    rows = []


    if not Path(root).exists():

        return rows


    for path in sorted(
        p
        for p in Path(root).rglob(
            "*"
        )
        if p.is_file()
    ):

        size = int(
            path.stat().st_size
        )


        row = {
            "repo_relative_path":
                str(
                    path.relative_to(
                        REPO
                    )
                ),

            "size_bytes":
                size,

            "suffix":
                path.suffix.lower(),

            "sha256":
                (
                    sha256_file(
                        path
                    )
                    if size
                    <=
                    max_hash_size
                    else
                    None
                ),
        }


        rows.append(
            row
        )


    return rows


# =============================================================================
# 2. DURABLE GIT STATE
# =============================================================================

banner(
    "STAGE26-8D0 :: DURABLE SCIENTIFIC STATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-8D0 HEAD."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Stage26-8D0."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if RUNTIME_AUDIT.exists():

    raise RuntimeError(
        "Stage26-8D0 transient audit already exists."
    )


# =============================================================================
# 3. STAGE26-8C2 SENSITIVITY CLOSURE IDENTITY
# =============================================================================

banner(
    "STAGE26-8D0 :: 4C3 SENSITIVITY CLOSURE"
)


sensitivity_checks = [
    (
        "8C1 sensitivity audit",
        SENS_AUDIT,
        EXPECTED_SENS_AUDIT_SHA256,
    ),

    (
        "8C2 anchor receipt",
        SENS_ANCHOR_RECEIPT,
        EXPECTED_SENS_ANCHOR_RECEIPT_SHA256,
    ),

    (
        "8C1 sensitivity manifest",
        SENS_MANIFEST,
        EXPECTED_SENS_MANIFEST_SHA256,
    ),
]


for label, path, expected in sensitivity_checks:

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:30s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Sensitivity closure identity mismatch: {label}"
        )


sensitivity_audit = safe_json(
    SENS_AUDIT
)


if sensitivity_audit[
    "status"
] != "PASS_PAIRED_4C3_IMPLEMENTATION_SENSITIVITY_AUDIT_COMPLETED":

    raise RuntimeError(
        "Stage26-4C3 sensitivity audit is not completed."
    )


if sensitivity_audit[
    "publication_policy"
][
    "historical_stage26_4c3"
] != "RETAIN_AS_ORIGINAL_MEASUREMENT":

    raise RuntimeError(
        "Unexpected historical 4C3 publication policy."
    )


if sensitivity_audit[
    "publication_policy"
][
    "V3"
] != "PAIRED_SENSITIVITY_ONLY_NOT_REPLACEMENT":

    raise RuntimeError(
        "Unexpected V3 publication policy."
    )


print(
    "\n4C3 sensitivity audit status: COMPLETED"
)

print(
    "Historical 4C3             : RETAINED"
)

print(
    "V3                         : SENSITIVITY ONLY"
)


# =============================================================================
# 4. STAGE26-7C CAPACITY IDENTITY
# =============================================================================

banner(
    "STAGE26-8D0 :: STAGE26-7C CAPACITY / SCALING IDENTITY"
)


capacity_checks = [
    (
        "7C results",
        CAPACITY_RESULTS,
        EXPECTED_CAPACITY_RESULTS_SHA256,
    ),

    (
        "7C receipt",
        CAPACITY_RECEIPT,
        EXPECTED_CAPACITY_RECEIPT_SHA256,
    ),

    (
        "7C manifest",
        CAPACITY_MANIFEST,
        EXPECTED_CAPACITY_MANIFEST_SHA256,
    ),
]


for label, path, expected in capacity_checks:

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:20s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Stage26-7C identity mismatch: {label}"
        )


capacity_results = safe_json(
    CAPACITY_RESULTS
)


# Search explicitly for prohibited/allowed state.
capacity_flat = flatten_json(
    capacity_results
)


capacity_text = json.dumps(
    capacity_results,
    sort_keys=True,
)


if "complete" not in capacity_text.lower():

    print(
        "\nNOTE: complete-E2E state is not encoded directly in this file;"
    )

    print(
        "      Stage26-5A remains the authoritative E2E closure."
    )


print(
    "\nStage26-7C remains component/capacity reporting only."
)


# =============================================================================
# 5. HISTORICAL STAGE26-4C3 IDENTITY
# =============================================================================

banner(
    "STAGE26-8D0 :: HISTORICAL STAGE26-4C3 IDENTITY"
)


for label, path, expected in [
    (
        "4C3 summary CSV",
        REP_4C3_SUMMARY_CSV,
        EXPECTED_REP_4C3_SUMMARY_CSV_SHA256,
    ),

    (
        "4C3 summary JSON",
        REP_4C3_SUMMARY_JSON,
        EXPECTED_REP_4C3_SUMMARY_JSON_SHA256,
    ),
]:

    actual = sha256_file(
        path
    )

    passed = (
        actual == expected
    )


    print(
        f"{label:22s} "
        f"{'PASS' if passed else 'FAIL'} "
        f"{actual}"
    )


    if not passed:

        raise RuntimeError(
            f"Historical Stage26-4C3 identity mismatch: {label}"
        )


# =============================================================================
# 6. INVENTORY ALL STAGE26 CHECKPOINT DIRECTORIES
# =============================================================================

banner(
    "STAGE26-8D0 :: STAGE26 CHECKPOINT INVENTORY"
)


if not RESULT_ROOT.is_dir():

    raise FileNotFoundError(
        RESULT_ROOT
    )


checkpoint_dirs = sorted(
    path
    for path in RESULT_ROOT.iterdir()
    if path.is_dir()
)


print(
    "Checkpoint directories:",
    len(
        checkpoint_dirs
    )
)


for path in checkpoint_dirs:

    print(
        " ",
        path.name
    )


# =============================================================================
# 7. CLASSIFY RELEVANT FINAL-CPU SOURCE FAMILIES
# =============================================================================

banner(
    "STAGE26-8D0 :: FINAL CPU SOURCE FAMILY RESOLUTION"
)


FAMILY_TOKENS = {
    "cold_start": [
        "cold",
        "stage26_1",
        "26_1",
    ],

    "warm_cpu": [
        "warm",
        "stage26_2",
        "26_2",
    ],

    "memory_cpu": [
        "memory",
        "26_3",
        "stage26_3",
    ],

    "raw_extraction": [
        "raw",
        "extraction",
        "4b3",
    ],

    "representation_4c3": [
        "4c3",
        "representation_timing",
    ],

    "e2e_closure": [
        "5a",
        "e2e",
        "end_to_end",
    ],

    "pareto": [
        "6d",
        "pareto",
    ],

    "corrected_uncertainty": [
        "6f1",
        "corrected",
        "uncertainty",
    ],

    "capacity_scaling": [
        "7c",
        "capacity",
        "scaling",
    ],

    "representation_sensitivity": [
        "8c1",
        "sensitivity",
    ],
}


family_dirs = {}


for family, tokens in FAMILY_TOKENS.items():

    matches = []


    for path in checkpoint_dirs:

        lower = path.name.lower()


        score = sum(
            token.lower()
            in
            lower
            for token in tokens
        )


        if score > 0:

            matches.append(
                {
                    "path":
                        path,

                    "score":
                        score,
                }
            )


    matches.sort(
        key=lambda row: (
            -row[
                "score"
            ],
            row[
                "path"
            ].name,
        )
    )


    family_dirs[
        family
    ] = [
        str(
            row[
                "path"
            ].relative_to(
                REPO
            )
        )
        for row in matches
    ]


    print(
        f"\n{family}:"
    )


    if matches:

        for row in matches[
            :20
        ]:

            print(
                f"  score={row['score']} "
                f"{row['path'].name}"
            )

    else:

        print(
            "  <no directory-name match>"
        )


# =============================================================================
# 8. RESOLVE EXACT STAGE26-6F1 CORRECTED UNCERTAINTY BY SHA
# =============================================================================

banner(
    "STAGE26-8D0 :: EXACT STAGE26-6F1 CORRECTED UNCERTAINTY RESOLUTION"
)


# Result tree files are generally small.
# Hash only <=20 MiB files to avoid touching unrelated large artifacts.

all_small_files = []


for path in RESULT_ROOT.rglob(
    "*"
):

    if (
        not path.is_file()
        or
        path.stat().st_size
        >
        20 * 1024 * 1024
    ):

        continue


    all_small_files.append(
        path
    )


hash_to_paths = defaultdict(
    list
)


for path in all_small_files:

    digest = sha256_file(
        path
    )

    hash_to_paths[
        digest
    ].append(
        path
    )


resolved_6f1 = {}


for label, expected_sha in EXPECTED_6F1_HASHES.items():

    paths = hash_to_paths.get(
        expected_sha,
        []
    )


    print(
        f"\n{label}"
    )

    print(
        "  expected SHA:",
        expected_sha
    )

    print(
        "  matches     :",
        len(
            paths
        )
    )


    for path in paths:

        print(
            "   ",
            path.relative_to(
                REPO
            )
        )


    if len(
        paths
    ) != 1:

        raise RuntimeError(
            f"Could not uniquely resolve Stage26-6F1 file: {label}"
        )


    resolved_6f1[
        label
    ] = {
        "repo_relative_path":
            str(
                paths[
                    0
                ].relative_to(
                    REPO
                )
            ),

        "sha256":
            expected_sha,
    }


print(
    "\nStage26-6F1 corrected uncertainty package: EXACTLY RESOLVED"
)


# =============================================================================
# 9. RAW EXTRACTION SUMMARY SHA RESOLUTION
# =============================================================================

banner(
    "STAGE26-8D0 :: RAW EXTRACTION SUMMARY RESOLUTION"
)


raw_summary_paths = hash_to_paths.get(
    EXPECTED_RAW_EXTRACTION_SUMMARY_SHA256,
    []
)


print(
    "Expected summary SHA:",
    EXPECTED_RAW_EXTRACTION_SUMMARY_SHA256
)

print(
    "Matches:",
    len(
        raw_summary_paths
    )
)


for path in raw_summary_paths:

    print(
        " ",
        path.relative_to(
            REPO
        )
    )


if len(
    raw_summary_paths
) != 1:

    raise RuntimeError(
        "Could not uniquely resolve Stage26-4B3 raw extraction summary."
    )


# =============================================================================
# 10. COLD-START PROVENANCE DISCOVERY
# =============================================================================

banner(
    "STAGE26-8D0 :: STAGE26-1 COLD-START UNCERTAINTY PROVENANCE AUDIT"
)


# We must NOT assume that historical cold-start confidence intervals are
# certified merely because Stage26-2 corrected uncertainty is certified.
#
# Locate cold-start-related files via directory/file content.

cold_candidate_files = []


for path in all_small_files:

    rel = str(
        path.relative_to(
            REPO
        )
    ).lower()


    name_hit = (
        "cold"
        in
        rel
        or
        "stage26_1"
        in
        rel
        or
        "26_1"
        in
        rel
    )


    content_hit = False


    if path.suffix.lower() in {
        ".json",
        ".csv",
        ".md",
        ".txt",
        ".py",
    }:

        try:

            text = path.read_text(
                encoding="utf-8",
                errors="ignore",
            )


            lower_text = text.lower()


            content_hit = (
                "cold-start"
                in
                lower_text
                or
                "cold_start"
                in
                lower_text
                or
                "cold start"
                in
                lower_text
            )

        except Exception:

            pass


    if (
        name_hit
        or
        content_hit
    ):

        cold_candidate_files.append(
            path
        )


cold_candidate_files = sorted(
    set(
        cold_candidate_files
    )
)


print(
    "Cold-start candidate files:",
    len(
        cold_candidate_files
    )
)


for path in cold_candidate_files[
    :100
]:

    print(
        " ",
        path.relative_to(
            REPO
        )
    )


# -----------------------------------------------------------------------------
# Extract ONLY explicitly recorded uncertainty/bootstrap provenance clues.
# -----------------------------------------------------------------------------

bootstrap_terms = [
    "bootstrap",
    "replicates",
    "seed",
    "rng",
    "pcg64",
    "default_rng",
    "percentile",
    "confidence",
    "interval",
    "ci",
]


cold_provenance_hits = []


for path in cold_candidate_files:

    if path.suffix.lower() != ".json":

        continue


    obj = safe_json(
        path
    )


    if obj is None:

        continue


    flattened = flatten_json(
        obj
    )


    for item in flattened:

        key_lower = item[
            "path"
        ].lower()

        value_text = str(
            item[
                "value"
            ]
        )

        value_lower = value_text.lower()


        matched = [
            term
            for term in bootstrap_terms
            if (
                term
                in
                key_lower
                or
                term
                in
                value_lower
            )
        ]


        if matched:

            cold_provenance_hits.append(
                {
                    "repo_relative_path":
                        str(
                            path.relative_to(
                                REPO
                            )
                        ),

                    "json_path":
                        item[
                            "path"
                        ],

                    "value":
                        item[
                            "value"
                        ],

                    "matched_terms":
                        matched,
                }
            )


print(
    "\nExplicit cold-start JSON uncertainty/provenance clues:",
    len(
        cold_provenance_hits
    )
)


for item in cold_provenance_hits[
    :200
]:

    print(
        f"\n{item['repo_relative_path']}"
    )

    print(
        " ",
        item[
            "json_path"
        ],
        "=",
        repr(
            item[
                "value"
            ]
        )
    )


# -----------------------------------------------------------------------------
# Conservative provenance evidence summary.
#
# This does NOT certify a CI automatically.
# It only asks whether the critical implementation details are explicitly
# present somewhere in the cold-start provenance.
# -----------------------------------------------------------------------------

cold_text = json.dumps(
    cold_provenance_hits,
    sort_keys=True,
).lower()


cold_evidence = {
    "bootstrap_term_present":
        (
            "bootstrap"
            in
            cold_text
        ),

    "replicate_count_present":
        (
            "replicates"
            in
            cold_text
            or
            "bootstrap_replicates"
            in
            cold_text
        ),

    "seed_present":
        (
            '"seed"'
            in
            cold_text
            or
            ".seed"
            in
            cold_text
        ),

    "rng_identity_present":
        (
            "pcg64"
            in
            cold_text
            or
            "default_rng"
            in
            cold_text
        ),

    "interval_or_percentile_method_present":
        (
            "percentile"
            in
            cold_text
            or
            "confidence"
            in
            cold_text
            or
            "interval"
            in
            cold_text
        ),
}


print(
    "\nCold-start explicit provenance evidence:"
)


for label, value in cold_evidence.items():

    print(
        f"  {label:40s}: "
        f"{'YES' if value else 'NO'}"
    )


# Publication decision remains conservative.
#
# We do NOT certify cold-start CIs here simply from scattered metadata.
# If implementation-level bootstrap identity cannot be reconstructed from
# exact committed provenance, publication uses cold-start point estimates
# only.

all_cold_core_fields_explicit = all(
    cold_evidence.values()
)


cold_start_publication_uncertainty_status = (
    "EXPLICIT_PROVENANCE_FIELDS_FOUND_REQUIRES_FINAL_IMPLEMENTATION_LINK_AUDIT"
    if
    all_cold_core_fields_explicit
    else
    "POINT_ESTIMATES_ONLY_UNLESS_SEPARATE_PROVENANCE_RECOVERY_SUCCEEDS"
)


print(
    "\nCold-start publication uncertainty status:"
)

print(
    " ",
    cold_start_publication_uncertainty_status
)


# =============================================================================
# 11. E2E CLOSURE DISCOVERY / AUDIT
# =============================================================================

banner(
    "STAGE26-8D0 :: COMPLETE-E2E CLOSURE AUDIT"
)


e2e_json_candidates = []


for path in all_small_files:

    if path.suffix.lower() != ".json":

        continue


    rel_lower = str(
        path.relative_to(
            REPO
        )
    ).lower()


    if (
        "5a"
        in
        rel_lower
        or
        "e2e"
        in
        rel_lower
        or
        "end_to_end"
        in
        rel_lower
    ):

        e2e_json_candidates.append(
            path
        )


e2e_hits = []


for path in e2e_json_candidates:

    obj = safe_json(
        path
    )


    if obj is None:

        continue


    text = json.dumps(
        obj,
        sort_keys=True,
    )


    lower = text.lower()


    if (
        "closed_no_valid_complete_e2e_measurement"
        in
        lower
        or
        "complete_e2e"
        in
        lower
        or
        "end-to-end"
        in
        lower
    ):

        e2e_hits.append(
            {
                "repo_relative_path":
                    str(
                        path.relative_to(
                            REPO
                        )
                    ),

                "sha256":
                    sha256_file(
                        path
                    ),

                "contains_closed_no_valid_complete_e2e_measurement":
                    (
                        "closed_no_valid_complete_e2e_measurement"
                        in
                        lower
                    ),

                "contains_missing_imputation_false":
                    (
                        "missing"
                        in
                        lower
                        and
                        "imput"
                        in
                        lower
                    ),
            }
        )


print(
    "E2E-relevant JSON candidates:",
    len(
        e2e_hits
    )
)


for item in e2e_hits[
    :50
]:

    print(
        "\n ",
        item[
            "repo_relative_path"
        ]
    )

    print(
        "   SHA256:",
        item[
            "sha256"
        ]
    )

    print(
        "   CLOSED_NO_VALID_COMPLETE_E2E_MEASUREMENT:",
        item[
            "contains_closed_no_valid_complete_e2e_measurement"
        ]
    )


if not any(
    item[
        "contains_closed_no_valid_complete_e2e_measurement"
    ]
    for item in e2e_hits
):

    raise RuntimeError(
        "Could not recover authoritative Stage26 complete-E2E closure."
    )


print(
    "\nComplete E2E publication claim: PROHIBITED"
)


# =============================================================================
# 12. PARETO SOURCE DISCOVERY
# =============================================================================

banner(
    "STAGE26-8D0 :: PARETO IMMUTABILITY SOURCE AUDIT"
)


pareto_candidates = []


for path in all_small_files:

    rel_lower = str(
        path.relative_to(
            REPO
        )
    ).lower()


    if (
        "pareto"
        in
        rel_lower
        or
        "6d"
        in
        rel_lower
    ):

        pareto_candidates.append(
            path
        )


print(
    "Pareto-related files:",
    len(
        pareto_candidates
    )
)


for path in sorted(
    pareto_candidates
)[
    :100
]:

    print(
        " ",
        path.relative_to(
            REPO
        )
    )


print(
    "\nFrozen publication rule:"
)

print(
    "  Stage26-6D point-estimate Pareto remains immutable."
)

print(
    "  Stage26-8 does NOT recompute the frontier."
)

print(
    "  NO cross-group Pareto frontier."
)


# =============================================================================
# 13. PUBLICATION ELIGIBILITY MATRIX
# =============================================================================

banner(
    "STAGE26-8D0 :: CPU PUBLICATION ELIGIBILITY MATRIX"
)


publication_matrix = [
    {
        "source":
            "STAGE26-1_COLD_START",

        "publication_role":
            "COLD_START_POINT_ESTIMATES",

        "point_estimates":
            "ALLOWED",

        "uncertainty":
            cold_start_publication_uncertainty_status,

        "notes":
            (
                "Cold-start CI provenance must remain separate from "
                "Stage26-2 corrected uncertainty certification."
            ),
    },

    {
        "source":
            "STAGE26-2_WARM_CPU",

        "publication_role":
            "ISOLATED_CPU_INFERENCE",

        "point_estimates":
            "ALLOWED",

        "uncertainty":
            "USE_STAGE26_6F1_CORRECTED_CERTIFIED_CI_ONLY",

        "notes":
            (
                "Do not publish historical Stage26-2 bootstrap intervals "
                "as certified."
            ),
    },

    {
        "source":
            "STAGE26-3_CPU_MEMORY",

        "publication_role":
            "CPU_MEMORY_AND_PACKAGE_SIZE",

        "point_estimates":
            "ALLOWED",

        "uncertainty":
            "NOT_APPLICABLE_UNLESS_ALREADY_FROZEN",

        "notes":
            (
                "Preserve resource-limit outcomes; no missing-cost imputation."
            ),
    },

    {
        "source":
            "STAGE26-4B3_RAW_EXTRACTION",

        "publication_role":
            "RAW_EXTRACTION_COMPONENT_ONLY",

        "point_estimates":
            "ALLOWED_DESCRIPTIVE",

        "uncertainty":
            "NO_P95_P99_FROM_FIVE_REPS",

        "notes":
            (
                "Do not call this CICFlowMeter 70-feature extraction. "
                "No complete-pipeline extrapolation."
            ),
    },

    {
        "source":
            "STAGE26-4C3_REPRESENTATION",

        "publication_role":
            "REPRESENTATION_COMPONENT_ONLY",

        "point_estimates":
            "HISTORICAL_4C3_RETAINED",

        "uncertainty":
            "NO_NEW_UNCERTAINTY_IN_STAGE26_8",

        "notes":
            (
                "Report Stage26-8 paired V2/V3 implementation-sensitivity "
                "evidence alongside historical measurement."
            ),
    },

    {
        "source":
            "STAGE26-5A_E2E",

        "publication_role":
            "AVAILABILITY_BOUNDARY",

        "point_estimates":
            "NO_COMPLETE_E2E_MEASUREMENT",

        "uncertainty":
            "NOT_APPLICABLE",

        "notes":
            (
                "A+B+C additive E2E prohibited; missing boundaries not imputed."
            ),
    },

    {
        "source":
            "STAGE26-6D_PARETO",

        "publication_role":
            "DESCRIPTIVE_POINT_ESTIMATE_FRONTIER",

        "point_estimates":
            "ALLOWED_IMMUTABLE",

        "uncertainty":
            "TIMING_COMPONENT_CI_SEPARATE",

        "notes":
            (
                "No PR-AUC bootstrap. No cross-group Pareto."
            ),
    },

    {
        "source":
            "STAGE26-6F1",

        "publication_role":
            "CORRECTED_CPU_TIMING_UNCERTAINTY",

        "point_estimates":
            "SOURCE_STAGE26_2",

        "uncertainty":
            "CERTIFIED",

        "notes":
            (
                "2000-replicate corrected timing intervals only; "
                "PR-AUC is not a Stage26 bootstrap target."
            ),
    },

    {
        "source":
            "STAGE26-7C",

        "publication_role":
            "CPU_CAPACITY_AND_SCALING",

        "point_estimates":
            "ALLOWED_COMPONENT_LEVEL",

        "uncertainty":
            "NO_DERIVED_RATIO_CI",

        "notes":
            (
                "Representation-vs-inference ratios are component-level, "
                "not pipeline bottlenecks."
            ),
    },

    {
        "source":
            "STAGE26-8C1",

        "publication_role":
            "4C3_IMPLEMENTATION_SENSITIVITY",

        "point_estimates":
            "ALLOWED_DESCRIPTIVE",

        "uncertainty":
            "NO_BOOTSTRAP_NO_HYPOTHESIS_TEST",

        "notes":
            (
                "Batch-dependent direction; V3 does not replace historical 4C3."
            ),
    },
]


for row in publication_matrix:

    print(
        "\n",
        row[
            "source"
        ],
        sep="",
    )

    print(
        "  role        :",
        row[
            "publication_role"
        ]
    )

    print(
        "  estimates   :",
        row[
            "point_estimates"
        ]
    )

    print(
        "  uncertainty :",
        row[
            "uncertainty"
        ]
    )

    print(
        "  notes       :",
        row[
            "notes"
        ]
    )


# =============================================================================
# 14. FINAL CPU CLOSURE CONSTRAINTS
# =============================================================================

banner(
    "STAGE26-8D0 :: FINAL CPU PUBLICATION CONSTRAINTS"
)


closure_constraints = {
    "GPU_allowed":
        False,

    "new_timing_allowed":
        False,

    "new_inference_allowed":
        False,

    "new_bootstrap_allowed":
        False,

    "PR_AUC_bootstrap_allowed":
        False,

    "historical_Stage26_2_CI_as_certified":
        False,

    "Stage26_6F1_corrected_CI_as_certified":
        True,

    "historical_4C3_replaced_by_V3":
        False,

    "4C3_sensitivity_audit_complete":
        True,

    "Stage26_7C_recompute_allowed":
        False,

    "Pareto_recompute_allowed":
        False,

    "cross_group_Pareto_allowed":
        False,

    "complete_E2E_claim_allowed":
        False,

    "missing_cost_imputation_allowed":
        False,

    "resource_limit_outcomes_must_be_preserved":
        True,

    "raw_extraction_full_pipeline_claim_allowed":
        False,

    "component_capacity_as_pipeline_bottleneck_allowed":
        False,
}


for key, value in closure_constraints.items():

    print(
        f"{key:50s}: "
        f"{value}"
    )


# =============================================================================
# 15. WRITE TRANSIENT AUDIT
# =============================================================================

banner(
    "STAGE26-8D0 :: WRITE TRANSIENT PUBLICATION-SOURCE AUDIT"
)


audit_payload = {
    "schema":
        "stage26_8d0_cpu_publication_source_audit_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "status":
        "PASS_READ_ONLY_CPU_PUBLICATION_SOURCE_AUDIT",

    "stage26_8c2": {
        "sensitivity_audit_sha256":
            EXPECTED_SENS_AUDIT_SHA256,

        "anchor_receipt_sha256":
            EXPECTED_SENS_ANCHOR_RECEIPT_SHA256,

        "manifest_sha256":
            EXPECTED_SENS_MANIFEST_SHA256,

        "sensitivity_audit_complete":
            True,

        "historical_4C3_retained":
            True,

        "V3_replacement":
            False,
    },

    "stage26_7c": {
        "results_sha256":
            EXPECTED_CAPACITY_RESULTS_SHA256,

        "receipt_sha256":
            EXPECTED_CAPACITY_RECEIPT_SHA256,

        "manifest_sha256":
            EXPECTED_CAPACITY_MANIFEST_SHA256,

        "recompute_allowed":
            False,
    },

    "stage26_6f1_corrected_uncertainty":
        resolved_6f1,

    "stage26_4b3_raw_extraction": {
        "summary_repo_relative_path":
            str(
                raw_summary_paths[
                    0
                ].relative_to(
                    REPO
                )
            ),

        "summary_sha256":
            EXPECTED_RAW_EXTRACTION_SUMMARY_SHA256,

        "role":
            "COMPONENT_ONLY_DESCRIPTIVE",
    },

    "stage26_4c3_historical": {
        "summary_csv_sha256":
            EXPECTED_REP_4C3_SUMMARY_CSV_SHA256,

        "summary_json_sha256":
            EXPECTED_REP_4C3_SUMMARY_JSON_SHA256,

        "retained":
            True,

        "sensitivity_audit_completed":
            True,
    },

    "cold_start_uncertainty_audit": {
        "candidate_file_count":
            len(
                cold_candidate_files
            ),

        "candidate_files": [
            str(
                path.relative_to(
                    REPO
                )
            )
            for path in cold_candidate_files
        ],

        "explicit_provenance_hits":
            cold_provenance_hits,

        "evidence":
            cold_evidence,

        "publication_uncertainty_status":
            cold_start_publication_uncertainty_status,

        "automatically_certified":
            False,
    },

    "e2e_closure_candidates":
        e2e_hits,

    "family_directory_resolution":
        family_dirs,

    "publication_eligibility_matrix":
        publication_matrix,

    "closure_constraints":
        closure_constraints,

    "scientific_state": {
        "timing_executed":
            False,

        "representation_executed":
            False,

        "model_loaded":
            False,

        "inference_performed":
            False,

        "bootstrap_executed":
            False,

        "new_CI_computed":
            False,

        "ratio_recomputed":
            False,

        "Stage26_7C_recomputed":
            False,

        "Pareto_recomputed":
            False,

        "PCAP_accessed":
            False,

        "labels_accessed":
            False,

        "GPU_used":
            False,

        "Git_modified":
            False,

        "publication_tables_generated":
            False,

        "publication_figures_generated":
            False,
    },

    "next":
        (
            "Use this audited source matrix to freeze and generate the final "
            "publication-facing Stage26 CPU tables/figures. If cold-start "
            "implementation-level CI provenance remains unresolved, report "
            "cold-start point estimates without uncertified confidence intervals."
        ),
}


atomic_json(
    RUNTIME_AUDIT,
    audit_payload,
)


audit_sha = sha256_file(
    RUNTIME_AUDIT
)


print(
    "Audit:"
)

print(
    " ",
    RUNTIME_AUDIT
)

print(
    "SHA256:"
)

print(
    " ",
    audit_sha
)


# =============================================================================
# 16. FINAL READ-ONLY GIT CLOSURE
# =============================================================================

banner(
    "STAGE26-8D0 CPU PUBLICATION-SOURCE AUDIT COMPLETE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed during Stage26-8D0."
    )


if final_remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed during Stage26-8D0."
    )


if final_status:

    raise RuntimeError(
        "Stage26-8D0 unexpectedly modified Git."
    )


print(
    "\nCORE PUBLICATION STATE:"
)

print(
    "  Stage26-2 point estimates        : ALLOWED"
)

print(
    "  Stage26-2 historical CIs         : NOT CERTIFIED"
)

print(
    "  Stage26-6F1 corrected CIs        : CERTIFIED"
)

print(
    "  Stage26-4B3 extraction           : COMPONENT ONLY"
)

print(
    "  Stage26-4C3 historical           : RETAINED"
)

print(
    "  Stage26-8C1 sensitivity          : COMPLETED / DESCRIPTIVE"
)

print(
    "  Stage26-5A complete E2E          : NOT AVAILABLE"
)

print(
    "  Stage26-6D Pareto                : IMMUTABLE"
)

print(
    "  Stage26-7C capacity              : COMPONENT LEVEL"
)

print(
    "  cold-start CI status             :"
)

print(
    "   ",
    cold_start_publication_uncertainty_status
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  new timing              : NO"
)

print(
    "  inference               : NO"
)

print(
    "  bootstrap               : NO"
)

print(
    "  new CI                  : NO"
)

print(
    "  Stage26-7C recompute    : NO"
)

print(
    "  Pareto recompute        : NO"
)

print(
    "  complete-E2E derivation : NO"
)

print(
    "  PCAP                    : NO"
)

print(
    "  labels                  : NO"
)

print(
    "  GPU                     : NO"
)

print(
    "  Git modified            : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Inspect this source audit."
)

print(
    "  Then freeze the final publication-facing CPU table/figure schema"
)

print(
    "  before generating any publication artifact."
)

print(
    "  GPU remains OFF."
)


STAGE26-8D0 :: DURABLE SCIENTIFIC STATE
Expected parent: fd4497e333ebeb72b1eb188ed64ca2cdd0f08567
Local HEAD     : fd4497e333ebeb72b1eb188ed64ca2cdd0f08567
origin/main    : fd4497e333ebeb72b1eb188ed64ca2cdd0f08567
Repo clean     : True

STAGE26-8D0 :: 4C3 SENSITIVITY CLOSURE
8C1 sensitivity audit          PASS 7daaa517e49ca99787b124855bb425abc89b0ef4522556a7500c69f9fd962af2
8C2 anchor receipt             PASS 45d6d5e5fbb1dfccc08dc64449b9ec1af1951799208770824cc1cc702991e8ad
8C1 sensitivity manifest       PASS 82af40d79f713013e9914b62aa6d0683944bc19ec2310050d2733be2ee244982

4C3 sensitivity audit status: COMPLETED
Historical 4C3             : RETAINED
V3                         : SENSITIVITY ONLY

STAGE26-8D0 :: STAGE26-7C CAPACITY / SCALING IDENTITY
7C results           PASS ee5a852fb43ce01c980e64ef83611a88808d3a11a56a078609467fb899e39fdf
7C receipt           PASS 30bfd4657a5a0218ce22a4fab1922ee26e07496c148809ec04c8059656b73ce4
7C manifest          PASS f090a992195e718b9ae569e681b3fd4b

In [14]:
# =============================================================================
# STAGE26-8D1
# COLD-START CI IMPLEMENTATION-LINK / PROVENANCE AUDIT
#
# DURABLE SCIENTIFIC PARENT:
#   fd4497e333ebeb72b1eb188ed64ca2cdd0f08567
#
# PURPOSE
# -------
# Stage26-8D0 established that cold-start uncertainty has explicit protocol
# metadata:
#
#   bootstrap enabled
#   2000 replicates
#   percentile 95% interval
#   numpy.random.default_rng_PCG64
#   seed 26042
#
# and that Stage26-1 stores cold-start bootstrap CI values.
#
# That is NOT, by itself, enough to certify publication-facing CIs.
#
# This audit asks the remaining question:
#
#   Can the exact implementation that generated the Stage26-1 cold-start
#   bootstrap intervals be recovered from committed provenance and linked
#   unambiguously to the Stage26-1 result package?
#
#
# CERTIFICATION REQUIRES ALL OF:
# --------------------------------
# 1. prospective Stage26 statistics protocol identity is recoverable;
# 2. Stage26-1 raw observations and summary are committed;
# 3. an exact committed/historical source implementing cold-start bootstrap
#    resampling is identifiable;
# 4. that implementation is linked to Stage26-1 by committed provenance,
#    source hash, receipt, manifest, or exact introducing commit;
# 5. implementation exposes the RNG construction/seed;
# 6. implementation exposes replicate count;
# 7. implementation exposes the resampling mechanism;
# 8. implementation exposes CI endpoint calculation;
# 9. RNG stream partitioning/order is statically recoverable rather than
#    guessed after the results were observed.
#
# IMPORTANT
# ---------
# This cell DOES NOT recompute a bootstrap.
# It DOES NOT test whether a guessed implementation reproduces stored CIs.
#
# If exact implementation provenance cannot be certified, the final policy is:
#
#   COLD-START POINT ESTIMATES ONLY
#
# rather than inventing or reconstructing uncertainty post hoc.
#
#
# READ ONLY WITH RESPECT TO GIT.
#
# Writes one TRANSIENT audit:
#
#   /kaggle/working/stage26_deployment_profiling/cpu_closure/
#       stage26_8d1_cold_start_ci_implementation_link_audit.json
#
#
# NO:
#   - timing
#   - model loading
#   - inference
#   - representation execution
#   - bootstrap execution
#   - CI recomputation
#   - random resampling
#   - Pareto recomputation
#   - Stage26-7C recomputation
#   - PCAP
#   - labels
#   - GPU
#   - Git modification
# =============================================================================

from __future__ import annotations

import ast
import hashlib
import json
import os
import re
import subprocess
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

RESULT_ROOT = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
)

EXPECTED_PARENT = (
    "fd4497e333ebeb72b1eb188ed64ca2cdd0f08567"
)


# -----------------------------------------------------------------------------
# Prior Stage26-8D0 transient audit
# -----------------------------------------------------------------------------

AUDIT_8D0 = Path(
    "/kaggle/working/stage26_deployment_profiling/"
    "cpu_closure/"
    "stage26_8d0_cpu_publication_source_audit.json"
)

EXPECTED_8D0_SHA256 = (
    "6b0207793eabe8435e14a7bd33a438c4a0e8f1e766d9bdf0e18f02db84b8d530"
)


# -----------------------------------------------------------------------------
# Stage26 protocol
# -----------------------------------------------------------------------------

MEASUREMENT_PROTOCOL = (
    RESULT_ROOT
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)


# -----------------------------------------------------------------------------
# Stage26-1 cold-start package
# -----------------------------------------------------------------------------

COLD_DIR = (
    RESULT_ROOT
    / "stage26_1_cpu_cold_start"
)

COLD_IMPLEMENTATION = (
    COLD_DIR
    / "stage26_1_cold_start_implementation.json"
)

COLD_MANIFEST = (
    COLD_DIR
    / "stage26_1_cold_start_package_manifest.json"
)

COLD_RAW_CSV = (
    COLD_DIR
    / "stage26_1_cold_start_raw.csv"
)

COLD_RAW_JSONL = (
    COLD_DIR
    / "stage26_1_cold_start_raw.jsonl"
)

COLD_RECEIPT = (
    COLD_DIR
    / "stage26_1_cold_start_receipt.json"
)

COLD_SCHEDULE = (
    COLD_DIR
    / "stage26_1_cold_start_schedule.json"
)

COLD_SUMMARY_CSV = (
    COLD_DIR
    / "stage26_1_cold_start_summary.csv"
)

COLD_SUMMARY_JSON = (
    COLD_DIR
    / "stage26_1_cold_start_summary.json"
)

COLD_WORKER = (
    COLD_DIR
    / "stage26_cold_start_worker.py"
)


# Historical durable commit from Stage26-1.
EXPECTED_STAGE26_1_COMMIT = (
    "46379b6d036008db4d60b056a66f4c01383e3298"
)


# -----------------------------------------------------------------------------
# Transient output
# -----------------------------------------------------------------------------

RUNTIME_AUDIT = Path(
    "/kaggle/working/stage26_deployment_profiling/"
    "cpu_closure/"
    "stage26_8d1_cold_start_ci_implementation_link_audit.json"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(text)

    print(
        "=" * 124
    )


def git(
    *args,
    check=True,
):

    p = subprocess.run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )

    return p.stdout.strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def atomic_json(
    path,
    payload,
):

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path)
        +
        ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )


    os.replace(
        tmp,
        path,
    )


def safe_json(path):

    try:

        return json.loads(
            Path(path).read_text(
                encoding="utf-8"
            )
        )

    except Exception:

        return None


def flatten_json(
    value,
    prefix="$",
    output=None,
):

    if output is None:

        output = []


    if isinstance(
        value,
        dict,
    ):

        for key, child in value.items():

            flatten_json(
                child,
                (
                    prefix
                    +
                    "."
                    +
                    str(
                        key
                    )
                ),
                output,
            )


    elif isinstance(
        value,
        list,
    ):

        for index, child in enumerate(
            value
        ):

            flatten_json(
                child,
                f"{prefix}[{index}]",
                output,
            )


    else:

        output.append(
            {
                "path":
                    prefix,

                "value":
                    value,
            }
        )


    return output


def source_context(
    lines,
    line_number,
    radius=5,
):

    start = max(
        1,
        line_number
        -
        radius,
    )

    end = min(
        len(
            lines
        ),
        line_number
        +
        radius,
    )


    return "\n".join(
        f"{i:6d}: {lines[i - 1]}"
        for i in range(
            start,
            end + 1,
        )
    )


# =============================================================================
# 2. DURABLE STATE
# =============================================================================

banner(
    "STAGE26-8D1 :: DURABLE SCIENTIFIC STATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage26-8D1 HEAD."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Stage26-8D1."
    )


if status:

    raise RuntimeError(
        "Repository must be clean."
    )


if RUNTIME_AUDIT.exists():

    raise RuntimeError(
        "Stage26-8D1 transient audit already exists."
    )


# =============================================================================
# 3. PREVIOUS AUDIT IDENTITY
# =============================================================================

banner(
    "STAGE26-8D1 :: PREVIOUS PUBLICATION-SOURCE AUDIT"
)


if not AUDIT_8D0.is_file():

    raise FileNotFoundError(
        AUDIT_8D0
    )


actual_8d0_sha = sha256_file(
    AUDIT_8D0
)


print(
    "Expected SHA:",
    EXPECTED_8D0_SHA256
)

print(
    "Actual SHA  :",
    actual_8d0_sha
)


if actual_8d0_sha != EXPECTED_8D0_SHA256:

    raise RuntimeError(
        "Stage26-8D0 transient audit identity mismatch."
    )


audit_8d0 = safe_json(
    AUDIT_8D0
)


if (
    audit_8d0[
        "cold_start_uncertainty_audit"
    ][
        "publication_uncertainty_status"
    ]
    !=
    "EXPLICIT_PROVENANCE_FIELDS_FOUND_REQUIRES_FINAL_IMPLEMENTATION_LINK_AUDIT"
):

    raise RuntimeError(
        "Stage26-8D0 does not request this implementation-link audit."
    )


print(
    "Stage26-8D0 handoff: PASS"
)


# =============================================================================
# 4. PROTOCOL IDENTITY + EXACT FROZEN STATISTICS FIELDS
# =============================================================================

banner(
    "STAGE26-8D1 :: FROZEN STATISTICS PROTOCOL"
)


protocol_sha = sha256_file(
    MEASUREMENT_PROTOCOL
)


print(
    "Expected protocol SHA:",
    EXPECTED_PROTOCOL_SHA256
)

print(
    "Actual protocol SHA  :",
    protocol_sha
)


if protocol_sha != EXPECTED_PROTOCOL_SHA256:

    raise RuntimeError(
        "Stage26 measurement protocol identity mismatch."
    )


protocol = safe_json(
    MEASUREMENT_PROTOCOL
)


statistics_protocol = protocol[
    "statistics_protocol"
]

bootstrap_protocol = statistics_protocol[
    "bootstrap"
]


protocol_fields = {
    "enabled":
        bootstrap_protocol[
            "enabled"
        ],

    "interval":
        bootstrap_protocol[
            "interval"
        ],

    "replicates":
        bootstrap_protocol[
            "replicates"
        ],

    "rng":
        bootstrap_protocol[
            "rng"
        ],

    "seed":
        bootstrap_protocol[
            "seed"
        ],

    "targets":
        statistics_protocol[
            "bootstrap_targets"
        ],
}


print(
    json.dumps(
        protocol_fields,
        indent=2,
        sort_keys=True,
    )
)


if protocol_fields[
    "enabled"
] is not True:

    raise RuntimeError(
        "Frozen bootstrap protocol is not enabled."
    )


if protocol_fields[
    "replicates"
] != 2000:

    raise RuntimeError(
        "Frozen bootstrap replicate count changed."
    )


if protocol_fields[
    "seed"
] != 26042:

    raise RuntimeError(
        "Frozen bootstrap seed changed."
    )


if protocol_fields[
    "rng"
] != "numpy.random.default_rng_PCG64":

    raise RuntimeError(
        "Frozen bootstrap RNG identity changed."
    )


if protocol_fields[
    "interval"
] != "PERCENTILE_95_PERCENT":

    raise RuntimeError(
        "Frozen bootstrap interval changed."
    )


# =============================================================================
# 5. EXACT STAGE26-1 PACKAGE INVENTORY
# =============================================================================

banner(
    "STAGE26-8D1 :: STAGE26-1 PACKAGE INVENTORY"
)


required_files = [
    COLD_IMPLEMENTATION,
    COLD_MANIFEST,
    COLD_RAW_CSV,
    COLD_RAW_JSONL,
    COLD_RECEIPT,
    COLD_SCHEDULE,
    COLD_SUMMARY_CSV,
    COLD_SUMMARY_JSON,
    COLD_WORKER,
]


cold_file_inventory = []


for path in required_files:

    if not path.is_file():

        raise FileNotFoundError(
            path
        )


    row = {
        "repo_relative_path":
            str(
                path.relative_to(
                    REPO
                )
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


    cold_file_inventory.append(
        row
    )


    print(
        f"{path.name:48s} "
        f"{row['size_bytes']:10,d} B "
        f"{row['sha256']}"
    )


# =============================================================================
# 6. VERIFY HISTORICAL STAGE26-1 COMMIT
# =============================================================================

banner(
    "STAGE26-8D1 :: HISTORICAL STAGE26-1 COMMIT"
)


commit_exists = (
    git(
        "cat-file",
        "-t",
        EXPECTED_STAGE26_1_COMMIT,
        check=False,
    )
    ==
    "commit"
)


print(
    "Expected Stage26-1 commit:",
    EXPECTED_STAGE26_1_COMMIT
)

print(
    "Commit exists:",
    commit_exists
)


if not commit_exists:

    raise RuntimeError(
        "Historical Stage26-1 commit unavailable."
    )


commit_subject = git(
    "log",
    "-1",
    "--pretty=%s",
    EXPECTED_STAGE26_1_COMMIT,
)


print(
    "Subject:",
    commit_subject
)


# Verify current Stage26-1 package files already existed at that anchor and
# determine whether their bytes remain identical today.

historical_identity = {}


for row in cold_file_inventory:

    rel = row[
        "repo_relative_path"
    ]


    old_blob = subprocess.run(
        [
            "git",
            "show",
            f"{EXPECTED_STAGE26_1_COMMIT}:{rel}",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )


    exists_at_commit = (
        old_blob.returncode
        ==
        0
    )


    if exists_at_commit:

        old_sha = hashlib.sha256(
            old_blob.stdout
        ).hexdigest()

        byte_identical = (
            old_sha
            ==
            row[
                "sha256"
            ]
        )

    else:

        old_sha = None

        byte_identical = False


    historical_identity[
        rel
    ] = {
        "exists_at_stage26_1_commit":
            exists_at_commit,

        "sha256_at_stage26_1_commit":
            old_sha,

        "sha256_current":
            row[
                "sha256"
            ],

        "byte_identical":
            byte_identical,
    }


    print(
        f"{'PASS' if byte_identical else 'INFO'} "
        f"{rel}"
    )

    print(
        "  existed at Stage26-1 commit:",
        exists_at_commit
    )

    print(
        "  historical SHA:",
        old_sha
    )

    print(
        "  current SHA   :",
        row[
            "sha256"
        ]
    )


# =============================================================================
# 7. INSPECT COMMITTED STAGE26-1 JSON PROVENANCE
# =============================================================================

banner(
    "STAGE26-8D1 :: STAGE26-1 COMMITTED PROVENANCE FIELDS"
)


json_paths = [
    COLD_IMPLEMENTATION,
    COLD_MANIFEST,
    COLD_RECEIPT,
    COLD_SCHEDULE,
    COLD_SUMMARY_JSON,
]


provenance_terms = [
    "bootstrap",
    "replicate",
    "seed",
    "rng",
    "pcg",
    "default_rng",
    "percentile",
    "quantile",
    "confidence",
    "interval",
    "source",
    "script",
    "implementation",
    "worker",
    "sha256",
    "random",
]


json_provenance_hits = []


for path in json_paths:

    obj = safe_json(
        path
    )


    if obj is None:

        continue


    flattened = flatten_json(
        obj
    )


    local_hits = []


    for item in flattened:

        haystack = (
            item["path"]
            + " "
            + str(item["value"])
        ).lower()


        matched = [
            term
            for term in provenance_terms
            if term.lower()
            in
            haystack
        ]


        if matched:

            record = {
                "repo_relative_path":
                    str(
                        path.relative_to(
                            REPO
                        )
                    ),

                "json_path":
                    item[
                        "path"
                    ],

                "value":
                    item[
                        "value"
                    ],

                "matched_terms":
                    matched,
            }


            json_provenance_hits.append(
                record
            )

            local_hits.append(
                record
            )


    print(
        f"\n{path.relative_to(REPO)}"
    )

    print(
        "  relevant fields:",
        len(
            local_hits
        )
    )


    for record in local_hits[
        :100
    ]:

        print(
            " ",
            record[
                "json_path"
            ],
            "=",
            repr(
                record[
                    "value"
            ]
        )
    )


# =============================================================================
# 8. SEARCH CURRENT TRACKED SOURCE FOR ACTUAL BOOTSTRAP IMPLEMENTATION
# =============================================================================

banner(
    "STAGE26-8D1 :: CURRENT TRACKED BOOTSTRAP IMPLEMENTATION SEARCH"
)


tracked = git(
    "ls-files"
).splitlines()


source_extensions = {
    ".py",
    ".ipynb",
}


source_candidates = []


for rel in tracked:

    path = (
        REPO
        /
        rel
    )


    if (
        not path.is_file()
        or
        path.suffix.lower()
        not in
        source_extensions
    ):

        continue


    if path.stat().st_size > 10 * 1024 * 1024:

        continue


    try:

        text = path.read_text(
            encoding="utf-8",
            errors="ignore",
        )

    except Exception:

        continue


    lower = text.lower()


    cold_related = (
        "cold_start"
        in
        lower
        or
        "cold-start"
        in
        lower
        or
        "cold start"
        in
        lower
        or
        "p50_bootstrap_ci95"
        in
        lower
    )


    bootstrap_related = (
        "bootstrap"
        in
        lower
        or
        "p50_bootstrap_ci95"
        in
        lower
    )


    has_rng = (
        "default_rng"
        in
        lower
        or
        "pcg64"
        in
        lower
        or
        "np.random"
        in
        lower
        or
        "numpy.random"
        in
        lower
    )


    has_resampling = (
        ".choice("
        in
        lower
        or
        ".integers("
        in
        lower
        or
        "resample"
        in
        lower
    )


    has_ci_operation = (
        "percentile("
        in
        lower
        or
        "quantile("
        in
        lower
    )


    if (
        cold_related
        and
        bootstrap_related
    ):

        source_candidates.append(
            {
                "repo_relative_path":
                    rel,

                "path":
                    path,

                "sha256":
                    sha256_file(
                        path
                    ),

                "has_rng":
                    has_rng,

                "has_resampling":
                    has_resampling,

                "has_ci_operation":
                    has_ci_operation,

                "qualifies_as_executable_bootstrap_candidate":
                    (
                        has_rng
                        and
                        has_resampling
                        and
                        has_ci_operation
                    ),
            }
        )


print(
    "Cold/bootstrap source candidates:",
    len(
        source_candidates
    )
)


for item in source_candidates:

    print(
        "\n",
        item[
            "repo_relative_path"
        ],
        sep="",
    )

    print(
        "  SHA256       :",
        item[
            "sha256"
        ]
    )

    print(
        "  RNG          :",
        item[
            "has_rng"
        ]
    )

    print(
        "  resampling   :",
        item[
            "has_resampling"
        ]
    )

    print(
        "  CI operation :",
        item[
            "has_ci_operation"
        ]
    )

    print(
        "  executable bootstrap candidate:",
        item[
            "qualifies_as_executable_bootstrap_candidate"
        ]
    )


qualified_current_sources = [
    item
    for item in source_candidates
    if item[
        "qualifies_as_executable_bootstrap_candidate"
    ]
]


# =============================================================================
# 9. SEARCH GIT HISTORY FOR LOST/REMOVED BOOTSTRAP SOURCE
# =============================================================================

banner(
    "STAGE26-8D1 :: GIT HISTORY IMPLEMENTATION SEARCH"
)


search_strings = [
    "p50_bootstrap_ci95_low_ms",
    "bootstrap_replicates",
    "default_rng(26042)",
]


history_hits = {}


for needle in search_strings:

    output = git(
        "log",
        "--all",
        "--oneline",
        "-S",
        needle,
        "--",
        ".",
        check=False,
    )


    lines = [
        line
        for line in output.splitlines()
        if line.strip()
    ]


    history_hits[
        needle
    ] = lines


    print(
        f"\n-S {needle!r}"
    )


    if lines:

        for line in lines[
            :50
        ]:

            print(
                " ",
                line
            )

    else:

        print(
            "  <no commit hits>"
        )


history_commits = []


for lines in history_hits.values():

    for line in lines:

        sha = line.split(
            maxsplit=1
        )[
            0
        ]


        if (
            re.fullmatch(
                r"[0-9a-fA-F]{7,40}",
                sha,
            )
            and
            sha not in history_commits
        ):

            history_commits.append(
                sha
            )


print(
    "\nUnique relevant history commits:",
    len(
        history_commits
    )
)


# =============================================================================
# 10. SEARCH RELEVANT HISTORICAL TREES FOR SOURCE FILES
# =============================================================================

banner(
    "STAGE26-8D1 :: HISTORICAL SOURCE RECOVERY"
)


historical_source_candidates = []


# Include exact Stage26-1 anchor plus every -S hit.
commits_to_scan = [
    EXPECTED_STAGE26_1_COMMIT,
    *history_commits,
]


dedup_commits = []


for commit in commits_to_scan:

    full = git(
        "rev-parse",
        commit,
        check=False,
    )


    if (
        re.fullmatch(
            r"[0-9a-f]{40}",
            full,
        )
        and
        full not in dedup_commits
    ):

        dedup_commits.append(
            full
        )


for commit in dedup_commits:

    names = git(
        "ls-tree",
        "-r",
        "--name-only",
        commit,
        check=False,
    ).splitlines()


    for rel in names:

        suffix = Path(
            rel
        ).suffix.lower()


        if suffix not in source_extensions:

            continue


        blob = subprocess.run(
            [
                "git",
                "show",
                f"{commit}:{rel}",
            ],
            cwd=REPO,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            check=False,
        )


        if blob.returncode != 0:

            continue


        if len(
            blob.stdout
        ) > 10 * 1024 * 1024:

            continue


        text = blob.stdout.decode(
            "utf-8",
            errors="ignore",
        )

        lower = text.lower()


        cold_related = (
            "cold_start"
            in
            lower
            or
            "cold-start"
            in
            lower
            or
            "cold start"
            in
            lower
            or
            "p50_bootstrap_ci95"
            in
            lower
        )


        bootstrap_related = (
            "bootstrap"
            in
            lower
            or
            "p50_bootstrap_ci95"
            in
            lower
        )


        has_rng = (
            "default_rng"
            in
            lower
            or
            "pcg64"
            in
            lower
            or
            "np.random"
            in
            lower
            or
            "numpy.random"
            in
            lower
        )


        has_resampling = (
            ".choice("
            in
            lower
            or
            ".integers("
            in
            lower
            or
            "resample"
            in
            lower
        )


        has_ci_operation = (
            "percentile("
            in
            lower
            or
            "quantile("
            in
            lower
        )


        if (
            cold_related
            and
            bootstrap_related
        ):

            digest = hashlib.sha256(
                blob.stdout
            ).hexdigest()


            key = (
                commit,
                rel,
                digest,
            )


            if not any(
                (
                    item[
                        "commit"
                    ],
                    item[
                        "repo_relative_path"
                    ],
                    item[
                        "sha256"
                    ],
                )
                ==
                key
                for item in historical_source_candidates
            ):

                historical_source_candidates.append(
                    {
                        "commit":
                            commit,

                        "repo_relative_path":
                            rel,

                        "sha256":
                            digest,

                        "text":
                            text,

                        "has_rng":
                            has_rng,

                        "has_resampling":
                            has_resampling,

                        "has_ci_operation":
                            has_ci_operation,

                        "qualifies_as_executable_bootstrap_candidate":
                            (
                                has_rng
                                and
                                has_resampling
                                and
                                has_ci_operation
                            ),
                    }
                )


print(
    "Historical cold/bootstrap source candidates:",
    len(
        historical_source_candidates
    )
)


for item in historical_source_candidates:

    print(
        "\nCommit:",
        item[
            "commit"
        ]
    )

    print(
        "Path  :",
        item[
            "repo_relative_path"
        ]
    )

    print(
        "SHA   :",
        item[
            "sha256"
        ]
    )

    print(
        "RNG/resample/CI:",
        item[
            "has_rng"
        ],
        item[
            "has_resampling"
        ],
        item[
            "has_ci_operation"
        ]
    )

    print(
        "Executable bootstrap candidate:",
        item[
            "qualifies_as_executable_bootstrap_candidate"
        ]
    )


qualified_historical_sources = [
    item
    for item in historical_source_candidates
    if item[
        "qualifies_as_executable_bootstrap_candidate"
    ]
]


# Deduplicate actual source bytes.
qualified_by_sha = {}


for item in [
    *qualified_current_sources,
    *qualified_historical_sources,
]:

    digest = item[
        "sha256"
    ]


    qualified_by_sha.setdefault(
        digest,
        []
    ).append(
        item
    )


print(
    "\nUnique qualifying implementation byte identities:",
    len(
        qualified_by_sha
    )
)


# =============================================================================
# 11. SEARCH COMMITTED PROVENANCE FOR QUALIFYING IMPLEMENTATION HASHES
# =============================================================================

banner(
    "STAGE26-8D1 :: IMPLEMENTATION HASH LINK AUDIT"
)


cold_provenance_texts = {}


for path in [
    COLD_IMPLEMENTATION,
    COLD_MANIFEST,
    COLD_RECEIPT,
    COLD_SCHEDULE,
    COLD_SUMMARY_JSON,
]:

    cold_provenance_texts[
        str(
            path.relative_to(
                REPO
            )
        )
    ] = path.read_text(
        encoding="utf-8",
        errors="ignore",
    )


hash_links = {}


for implementation_sha in qualified_by_sha:

    linked_files = []


    for rel, text in cold_provenance_texts.items():

        if implementation_sha in text:

            linked_files.append(
                rel
            )


    hash_links[
        implementation_sha
    ] = linked_files


    print(
        "\nImplementation SHA:",
        implementation_sha
    )

    print(
        "Linked by exact SHA in Stage26-1 provenance:",
        len(
            linked_files
        )
    )


    for rel in linked_files:

        print(
            " ",
            rel
        )


# =============================================================================
# 12. SOURCE SEMANTICS AUDIT
# =============================================================================

banner(
    "STAGE26-8D1 :: QUALIFYING SOURCE SEMANTICS"
)


source_semantics = {}


for implementation_sha, instances in qualified_by_sha.items():

    # Any instance with these exact bytes is equivalent for static semantics.
    representative = instances[
        0
    ]


    if "text" in representative:

        text = representative[
            "text"
        ]

    else:

        text = Path(
            representative[
                "path"
            ]
        ).read_text(
            encoding="utf-8",
            errors="ignore",
        )


    lines = text.splitlines()


    relevant_patterns = {
        "bootstrap":
            re.compile(
                r"bootstrap",
                re.I,
            ),

        "rng":
            re.compile(
                r"default_rng|PCG64|np\.random|numpy\.random",
                re.I,
            ),

        "seed":
            re.compile(
                r"26042|seed",
                re.I,
            ),

        "replicates":
            re.compile(
                r"2000|replicate",
                re.I,
            ),

        "resampling":
            re.compile(
                r"\.choice\s*\(|\.integers\s*\(|resample",
                re.I,
            ),

        "ci":
            re.compile(
                r"percentile\s*\(|quantile\s*\(|2\.5|97\.5|95",
                re.I,
            ),

        "summary_field":
            re.compile(
                r"p50_bootstrap_ci95|p95_bootstrap_ci95",
                re.I,
            ),
    }


    hit_lines = defaultdict(
        list
    )


    for number, line in enumerate(
        lines,
        start=1,
    ):

        for label, pattern in relevant_patterns.items():

            if pattern.search(
                line
            ):

                hit_lines[
                    label
                ].append(
                    number
                )


    # AST evidence where possible.
    default_rng_calls = []

    resample_calls = []

    percentile_calls = []

    enclosing_loop_headers = []


    try:

        tree = ast.parse(
            text
        )


        parent = {}


        for node in ast.walk(
            tree
        ):

            for child in ast.iter_child_nodes(
                node
            ):

                parent[
                    child
                ] = node


        for node in ast.walk(
            tree
        ):

            if not isinstance(
                node,
                ast.Call,
            ):

                continue


            try:

                call_name = ast.unparse(
                    node.func
                )

            except Exception:

                call_name = ""


            try:

                call_text = ast.unparse(
                    node
                )

            except Exception:

                call_text = call_name


            lower_name = call_name.lower()


            if "default_rng" in lower_name:

                record = {
                    "line":
                        node.lineno,

                    "call":
                        call_text,

                    "enclosing_loops":
                        [],
                }


                ancestor = parent.get(
                    node
                )


                while ancestor is not None:

                    if isinstance(
                        ancestor,
                        (
                            ast.For,
                            ast.While,
                        ),
                    ):

                        try:

                            header = ast.unparse(
                                ancestor
                            ).splitlines()[
                                0
                            ]

                        except Exception:

                            header = (
                                f"{type(ancestor).__name__} "
                                f"line {ancestor.lineno}"
                            )


                        record[
                            "enclosing_loops"
                        ].append(
                            {
                                "line":
                                    ancestor.lineno,

                                "header":
                                    header,
                            }
                        )


                    ancestor = parent.get(
                        ancestor
                    )


                default_rng_calls.append(
                    record
                )


            if (
                lower_name.endswith(
                    ".choice"
                )
                or
                lower_name.endswith(
                    ".integers"
                )
            ):

                resample_calls.append(
                    {
                        "line":
                            node.lineno,

                        "call":
                            call_text,
                    }
                )


            if (
                lower_name.endswith(
                    ".percentile"
                )
                or
                lower_name.endswith(
                    ".quantile"
                )
            ):

                percentile_calls.append(
                    {
                        "line":
                            node.lineno,

                        "call":
                            call_text,
                    }
                )


    except SyntaxError:

        pass


    explicit_seed_26042 = bool(
        re.search(
            r"default_rng\s*\(\s*26042\s*\)",
            text,
        )
    )


    explicit_replicates_2000 = bool(
        re.search(
            r"\b2000\b",
            text,
        )
    )


    explicit_percentile_endpoints = (
        (
            "2.5"
            in
            text
            and
            "97.5"
            in
            text
        )
        or
        "PERCENTILE_95_PERCENT"
        in
        text
    )


    has_exact_summary_fields = (
        "p50_bootstrap_ci95_low_ms"
        in
        text
        and
        "p50_bootstrap_ci95_high_ms"
        in
        text
        and
        "p95_bootstrap_ci95_low_ms"
        in
        text
        and
        "p95_bootstrap_ci95_high_ms"
        in
        text
    )


    source_semantics[
        implementation_sha
    ] = {
        "exact_hash_links":
            hash_links.get(
                implementation_sha,
                [],
            ),

        "default_rng_calls":
            default_rng_calls,

        "resample_calls":
            resample_calls,

        "percentile_calls":
            percentile_calls,

        "explicit_default_rng_seed_26042":
            explicit_seed_26042,

        "explicit_replicates_2000":
            explicit_replicates_2000,

        "explicit_percentile_95_endpoints":
            explicit_percentile_endpoints,

        "exact_cold_start_CI_summary_fields_present":
            has_exact_summary_fields,

        "hit_lines":
            dict(
                hit_lines
            ),
    }


    print(
        "\n"
        +
        "-" * 124
    )

    print(
        "IMPLEMENTATION SHA:",
        implementation_sha
    )

    print(
        "-" * 124
    )

    print(
        "exact Stage26-1 hash links:",
        hash_links.get(
            implementation_sha,
            [],
        )
    )

    print(
        "explicit default_rng(26042):",
        explicit_seed_26042
    )

    print(
        "explicit 2000 replicates:",
        explicit_replicates_2000
    )

    print(
        "explicit percentile-95 endpoints:",
        explicit_percentile_endpoints
    )

    print(
        "cold CI output fields present:",
        has_exact_summary_fields
    )


    print(
        "\ndefault_rng calls:"
    )


    for record in default_rng_calls:

        print(
            " ",
            record
        )


    print(
        "\nresample calls:"
    )


    for record in resample_calls:

        print(
            " ",
            record
        )


    print(
        "\npercentile/quantile calls:"
    )


    for record in percentile_calls:

        print(
            " ",
            record
        )


    # Print compact contexts only around the most relevant code.
    critical_lines = sorted(
        set(
            hit_lines[
                "rng"
            ]
            +
            hit_lines[
                "resampling"
            ]
            +
            hit_lines[
                "ci"
            ]
            +
            hit_lines[
                "summary_field"
            ]
        )
    )


    printed_ranges = []


    for number in critical_lines[
        :40
    ]:

        start = max(
            1,
            number - 4,
        )

        end = min(
            len(
                lines
            ),
            number + 6,
        )


        if any(
            start >= old_start
            and
            end <= old_end
            for old_start, old_end in printed_ranges
        ):

            continue


        printed_ranges.append(
            (
                start,
                end,
            )
        )


        print(
            "\n"
            +
            source_context(
                lines,
                number,
                radius=5,
            )
        )


# =============================================================================
# 13. CHECK WHETHER STAGE26-1 WORKER ITSELF COMPUTES BOOTSTRAP
# =============================================================================

banner(
    "STAGE26-8D1 :: COLD WORKER ROLE"
)


worker_text = COLD_WORKER.read_text(
    encoding="utf-8",
    errors="ignore",
)

worker_lower = worker_text.lower()


worker_bootstrap_role = {
    "contains_bootstrap":
        (
            "bootstrap"
            in
            worker_lower
        ),

    "contains_default_rng":
        (
            "default_rng"
            in
            worker_lower
        ),

    "contains_resampling":
        (
            ".choice("
            in
            worker_lower
            or
            ".integers("
            in
            worker_lower
        ),

    "contains_percentile":
        (
            "percentile("
            in
            worker_lower
            or
            "quantile("
            in
            worker_lower
        ),
}


for label, value in worker_bootstrap_role.items():

    print(
        f"{label:28s}: "
        f"{value}"
    )


# =============================================================================
# 14. FINAL IMPLEMENTATION-LINK CLASSIFICATION
# =============================================================================

banner(
    "STAGE26-8D1 :: COLD-START CI CERTIFICATION VERDICT"
)


# An implementation is publication-certifiable only if its exact bytes are
# uniquely identified and the critical mechanics are explicit.
#
# A source does NOT become certified merely because it happens to contain
# similar bootstrap code somewhere in repository history.


certifiable_candidates = []


for implementation_sha, semantics in source_semantics.items():

    linked = (
        len(
            semantics[
                "exact_hash_links"
            ]
        )
        >
        0
    )


    mechanics_complete = (
        len(
            semantics[
                "default_rng_calls"
            ]
        )
        >
        0
        and
        len(
            semantics[
                "resample_calls"
            ]
        )
        >
        0
        and
        len(
            semantics[
                "percentile_calls"
            ]
        )
        >
        0
        and
        semantics[
            "explicit_default_rng_seed_26042"
        ]
        and
        semantics[
            "explicit_replicates_2000"
        ]
        and
        semantics[
            "explicit_percentile_95_endpoints"
        ]
        and
        semantics[
            "exact_cold_start_CI_summary_fields_present"
        ]
    )


    if (
        linked
        and
        mechanics_complete
    ):

        certifiable_candidates.append(
            implementation_sha
        )


print(
    "Unique qualifying source byte identities:",
    len(
        qualified_by_sha
    )
)

print(
    "Hash-linked + mechanics-complete candidates:",
    len(
        certifiable_candidates
    )
)


for sha in certifiable_candidates:

    print(
        " ",
        sha
    )


# Even with complete mechanics, RNG stream partitioning must be recoverable
# from the exact source rather than inferred.
#
# Here we treat stream ordering as statically recoverable only when:
#
#   - there is exactly one certifiable implementation byte identity; and
#   - its default_rng call(s) can be located in source; and
#   - the implementation itself contains both the resampling and CI-writing
#     logic.
#
# The exact code contexts printed above remain part of the audit evidence.

rng_stream_order_statically_recoverable = (
    len(
        certifiable_candidates
    )
    ==
    1
)


if len(
    certifiable_candidates
) == 1:

    final_classification = (
        "COLD_START_CI_IMPLEMENTATION_PROVENANCE_CERTIFIED"
    )

    publication_policy = (
        "COLD_START_POINT_ESTIMATES_AND_COMMITTED_BOOTSTRAP_CIS_ALLOWED"
    )

    reason = (
        "Exactly one committed/historical bootstrap implementation byte "
        "identity is both linked to Stage26-1 provenance and exposes the "
        "frozen RNG seed, replicate count, resampling operation, percentile "
        "interval mechanics, and cold-start CI output fields."
    )


else:

    final_classification = (
        "COLD_START_CI_IMPLEMENTATION_PROVENANCE_NOT_CERTIFIED"
    )

    publication_policy = (
        "COLD_START_POINT_ESTIMATES_ONLY"
    )

    reason = (
        "The exact implementation that generated the stored cold-start "
        "bootstrap intervals cannot be uniquely and completely linked from "
        "committed Stage26-1 provenance. No post-hoc bootstrap reconstruction "
        "will be used."
    )


print(
    "Classification:"
)

print(
    " ",
    final_classification
)

print(
    "\nPublication policy:"
)

print(
    " ",
    publication_policy
)

print(
    "\nRNG stream/order statically recoverable:"
)

print(
    " ",
    rng_stream_order_statically_recoverable
)

print(
    "\nReason:"
)

print(
    " ",
    reason
)


# =============================================================================
# 15. IMPORTANT NON-COMPUTATION ASSERTIONS
# =============================================================================

banner(
    "STAGE26-8D1 :: NON-COMPUTATION ASSERTIONS"
)


scientific_state = {
    "bootstrap_executed":
        False,

    "bootstrap_CI_recomputed":
        False,

    "random_resampling_executed":
        False,

    "cold_start_timing_reexecuted":
        False,

    "model_loaded":
        False,

    "inference_performed":
        False,

    "representation_executed":
        False,

    "Stage26_7C_recomputed":
        False,

    "Pareto_recomputed":
        False,

    "PCAP_accessed":
        False,

    "labels_accessed":
        False,

    "GPU_used":
        False,

    "Git_modified":
        False,
}


for key, value in scientific_state.items():

    print(
        f"{key:40s}: "
        f"{value}"
    )


# =============================================================================
# 16. WRITE TRANSIENT AUDIT
# =============================================================================

banner(
    "STAGE26-8D1 :: WRITE TRANSIENT AUDIT"
)


audit_payload = {
    "schema":
        "stage26_8d1_cold_start_ci_implementation_link_audit_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "status":
        "PASS_READ_ONLY_COLD_START_CI_IMPLEMENTATION_LINK_AUDIT",

    "stage26_8d0": {
        "sha256":
            EXPECTED_8D0_SHA256,

        "handoff_status":
            (
                "EXPLICIT_PROVENANCE_FIELDS_FOUND_"
                "REQUIRES_FINAL_IMPLEMENTATION_LINK_AUDIT"
            ),
    },

    "measurement_protocol": {
        "repo_relative_path":
            str(
                MEASUREMENT_PROTOCOL.relative_to(
                    REPO
                )
            ),

        "sha256":
            protocol_sha,

        "bootstrap":
            protocol_fields,
    },

    "stage26_1_commit": {
        "commit":
            EXPECTED_STAGE26_1_COMMIT,

        "subject":
            commit_subject,

        "package_historical_identity":
            historical_identity,
    },

    "stage26_1_package_inventory":
        cold_file_inventory,

    "stage26_1_json_provenance_hits":
        json_provenance_hits,

    "current_source_candidates": [
        {
            key:
                value
            for key, value in item.items()
            if key not in {
                "path"
            }
        }
        for item in source_candidates
    ],

    "historical_source_candidates": [
        {
            key:
                value
            for key, value in item.items()
            if key != "text"
        }
        for item in historical_source_candidates
    ],

    "qualified_source_byte_identities": {
        sha: [
            {
                key:
                    value
                for key, value in item.items()
                if key not in {
                    "path",
                    "text",
                }
            }
            for item in instances
        ]
        for sha, instances in qualified_by_sha.items()
    },

    "implementation_hash_links":
        hash_links,

    "source_semantics":
        source_semantics,

    "cold_worker_bootstrap_role":
        worker_bootstrap_role,

    "certifiable_candidate_sha256":
        certifiable_candidates,

    "rng_stream_order_statically_recoverable":
        rng_stream_order_statically_recoverable,

    "final_classification":
        final_classification,

    "publication_policy":
        publication_policy,

    "reason":
        reason,

    "scientific_state":
        scientific_state,

    "next":
        (
            "Freeze final Stage26 CPU publication table/figure schema using "
            "this verdict. Do not recompute cold-start bootstrap uncertainty."
        ),
}


atomic_json(
    RUNTIME_AUDIT,
    audit_payload,
)


audit_sha = sha256_file(
    RUNTIME_AUDIT
)


print(
    "Audit:"
)

print(
    " ",
    RUNTIME_AUDIT
)

print(
    "SHA256:"
)

print(
    " ",
    audit_sha
)


# =============================================================================
# 17. FINAL READ-ONLY GIT CLOSURE
# =============================================================================

banner(
    "STAGE26-8D1 COMPLETE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_remote = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head
)

print(
    "origin/main:",
    final_remote
)

print(
    "Repo clean :",
    final_status == ""
)


if final_head != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed during Stage26-8D1."
    )


if final_remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed during Stage26-8D1."
    )


if final_status:

    raise RuntimeError(
        "Stage26-8D1 unexpectedly modified Git."
    )


print(
    "\nFINAL COLD-START UNCERTAINTY VERDICT:"
)

print(
    " ",
    final_classification
)


print(
    "\nPUBLICATION POLICY:"
)

print(
    " ",
    publication_policy
)


print(
    "\nNO POST-HOC RECALCULATION:"
)

print(
    "  bootstrap recomputed : NO"
)

print(
    "  CIs recomputed       : NO"
)

print(
    "  random resampling    : NO"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  timing          : NO"
)

print(
    "  inference       : NO"
)

print(
    "  model loading   : NO"
)

print(
    "  Stage26-7C      : UNCHANGED"
)

print(
    "  Pareto          : UNCHANGED"
)

print(
    "  PCAP            : NO"
)

print(
    "  labels          : NO"
)

print(
    "  GPU             : NO"
)

print(
    "  Git modified    : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Freeze the final Stage26 CPU publication table/figure schema"
)

print(
    "  using this exact cold-start uncertainty verdict."
)

print(
    "  GPU remains OFF."
)


STAGE26-8D1 :: DURABLE SCIENTIFIC STATE
Expected parent: fd4497e333ebeb72b1eb188ed64ca2cdd0f08567
Local HEAD     : fd4497e333ebeb72b1eb188ed64ca2cdd0f08567
origin/main    : fd4497e333ebeb72b1eb188ed64ca2cdd0f08567
Repo clean     : True

STAGE26-8D1 :: PREVIOUS PUBLICATION-SOURCE AUDIT
Expected SHA: 6b0207793eabe8435e14a7bd33a438c4a0e8f1e766d9bdf0e18f02db84b8d530
Actual SHA  : 6b0207793eabe8435e14a7bd33a438c4a0e8f1e766d9bdf0e18f02db84b8d530
Stage26-8D0 handoff: PASS

STAGE26-8D1 :: FROZEN STATISTICS PROTOCOL
Expected protocol SHA: d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
Actual protocol SHA  : d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
{
  "enabled": true,
  "interval": "PERCENTILE_95_PERCENT",
  "replicates": 2000,
  "rng": "numpy.random.default_rng_PCG64",
  "seed": 26042,
  "targets": [
    "p50_latency",
    "p95_latency",
    "p99_latency_when_n_gte_100",
    "median_throughput"
  ]
}

STAGE26-8D1 :: STAGE26-1 PACKAGE INVENTORY
stage26

In [15]:
# =============================================================================
# STAGE26-8D2
# FINAL CPU PUBLICATION TABLE / FIGURE SCHEMA FREEZE
#
# SCIENTIFIC PARENT:
#   fd4497e333ebeb72b1eb188ed64ca2cdd0f08567
#
# PURPOSE
# -------
# Freeze publication-facing CPU table/figure structure BEFORE generating any
# publication artifacts.
#
# This cell:
#   - verifies the Stage26-8D1 cold-start provenance verdict
#   - freezes exact table/figure families, column order, source anchors,
#     wording boundaries, and prohibited claims
#   - writes a durable schema lock + receipt + manifest
#   - commits and pushes the lock
#   - verifies remote bytes and a clean repository
#
# It DOES NOT:
#   - generate tables or figures
#   - recompute statistics
#   - run bootstrap
#   - run timing/inference/models
#   - access PCAP/labels
#   - recompute Pareto or Stage26-7C
#   - use GPU
# =============================================================================

from __future__ import annotations

import hashlib
import json
import os
import stat
import subprocess
import tempfile
from datetime import datetime, timezone
from pathlib import Path


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

RESULT_ROOT = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
)

EXPECTED_PARENT = (
    "fd4497e333ebeb72b1eb188ed64ca2cdd0f08567"
)

EXPECTED_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

EXPECTED_8D1_SHA256 = (
    "c90c80936c522df42ba387982bd51b1dfb6548b968a1616ee28d74e7e2800d00"
)

EXPECTED_8D1_VERDICT = (
    "COLD_START_CI_IMPLEMENTATION_PROVENANCE_NOT_CERTIFIED"
)

EXPECTED_8D1_POLICY = (
    "COLD_START_POINT_ESTIMATES_ONLY"
)


PROTOCOL = (
    RESULT_ROOT
    / "stage26_0_protocol_lock"
    / "measurement_protocol.json"
)


AUDIT_8D1 = Path(
    "/kaggle/working/stage26_deployment_profiling/"
    "cpu_closure/"
    "stage26_8d1_cold_start_ci_implementation_link_audit.json"
)


OUT_DIR = (
    RESULT_ROOT
    / "stage26_8d2_cpu_publication_schema_lock"
)

SCHEMA_PATH = (
    OUT_DIR
    / "stage26_8d2_cpu_publication_schema.json"
)

RECEIPT_PATH = (
    OUT_DIR
    / "stage26_8d2_cpu_publication_schema_receipt.json"
)

MANIFEST_PATH = (
    OUT_DIR
    / "stage26_8d2_cpu_publication_schema_manifest.json"
)


COMMIT_MESSAGE = (
    "stage26: freeze CPU publication schema"
)


SOURCE_ANCHORS = {
    "cold_start":
        "46379b6d036008db4d60b056a66f4c01383e3298",

    "warm_cpu_point_estimates":
        "7ffdba3f4ca4ea5cc53097d62aaf27957009b9f6",

    "cpu_memory_package":
        "347d93f21d454cc5bda2c45889c890c67cdf0ecc",

    "raw_extraction_component":
        "55aba2e1cc08385659479c6275234dc23b11d231",

    "representation_historical_4c3":
        "aee59fabab2465cca7c80904279bb6d3ef23894f",

    "complete_e2e_closure":
        "92495eac2c9202973ff4d889e3c12669c06e9ef1",

    "pareto_point_estimate":
        "ff9d329785c6cd30d273729356f402e59dc4e844",

    "corrected_warm_cpu_uncertainty_6f1":
        "a484148cd8ea7d7c89604d3a015f73bd6f1bc81a",

    "capacity_scaling_7c":
        "9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3",

    "representation_sensitivity_protocol":
        "973e0c479ce91ad21e92ed5db11585f26a4c49ec",

    "representation_sensitivity_results":
        "fd4497e333ebeb72b1eb188ed64ca2cdd0f08567",
}


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 120
    )

    print(
        text
    )

    print(
        "=" * 120
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
    env=None,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
        env=env,
    )

    if (
        check
        and
        p.returncode != 0
    ):

        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(map(str, cmd))}\n"
            f"{p.stdout}"
        )

    return p.stdout.strip()


def git(
    *args,
    check=True,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        check=check,
        env=env,
    )


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8
                * 1024
                * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def atomic_json(
    path,
    payload,
):

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path)
        +
        ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def read_json(path):

    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as f:

        return json.load(
            f
        )


def git_blob_bytes(
    ref,
    rel_path,
):

    p = subprocess.run(
        [
            "git",
            "show",
            f"{ref}:{rel_path}",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            f"Could not read {rel_path} from {ref}:\n"
            +
            p.stderr.decode(
                "utf-8",
                errors="replace",
            )
        )

    return p.stdout


# =============================================================================
# 2. DURABLE STATE GATE
# =============================================================================

banner(
    "STAGE26-8D2 :: DURABLE STATE GATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected local HEAD before Stage26-8D2."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Stage26-8D2."
    )


if status:

    raise RuntimeError(
        "Repository must be clean before Stage26-8D2."
    )


if OUT_DIR.exists():

    raise RuntimeError(
        "Stage26-8D2 output directory already exists:\n"
        f"{OUT_DIR}\n\n"
        "Do not overwrite a publication-schema freeze."
    )


# =============================================================================
# 3. VERIFY PROTOCOL + STAGE26-8D1
# =============================================================================

banner(
    "STAGE26-8D2 :: INPUT IDENTITY"
)


if not PROTOCOL.is_file():

    raise FileNotFoundError(
        PROTOCOL
    )


protocol_sha = sha256_file(
    PROTOCOL
)


print(
    "Protocol expected:",
    EXPECTED_PROTOCOL_SHA256
)

print(
    "Protocol actual  :",
    protocol_sha
)


if protocol_sha != EXPECTED_PROTOCOL_SHA256:

    raise RuntimeError(
        "Measurement protocol SHA mismatch."
    )


if not AUDIT_8D1.is_file():

    raise FileNotFoundError(
        AUDIT_8D1
    )


audit_8d1_sha = sha256_file(
    AUDIT_8D1
)


print(
    "8D1 expected SHA :",
    EXPECTED_8D1_SHA256
)

print(
    "8D1 actual SHA   :",
    audit_8d1_sha
)


if audit_8d1_sha != EXPECTED_8D1_SHA256:

    raise RuntimeError(
        "Stage26-8D1 audit SHA mismatch."
    )


audit_8d1 = read_json(
    AUDIT_8D1
)


verdict = audit_8d1.get(
    "final_classification"
)

cold_policy = audit_8d1.get(
    "publication_policy"
)


print(
    "8D1 verdict       :",
    verdict
)

print(
    "Cold-start policy :",
    cold_policy
)


if verdict != EXPECTED_8D1_VERDICT:

    raise RuntimeError(
        "Unexpected Stage26-8D1 verdict."
    )


if cold_policy != EXPECTED_8D1_POLICY:

    raise RuntimeError(
        "Unexpected Stage26-8D1 publication policy."
    )


# -----------------------------------------------------------------------------
# Verify every scientific anchor exists.
# -----------------------------------------------------------------------------

print(
    "\nSource anchors:"
)


for label, commit in SOURCE_ANCHORS.items():

    kind = git(
        "cat-file",
        "-t",
        commit,
        check=False,
    )

    print(
        f"{label:42s} "
        f"{commit} "
        f"{kind}"
    )

    if kind != "commit":

        raise RuntimeError(
            "Missing source anchor commit: "
            f"{label} -> {commit}"
        )


# =============================================================================
# 4. FREEZE PUBLICATION SCHEMA
# =============================================================================

banner(
    "STAGE26-8D2 :: FREEZE PUBLICATION TABLE / FIGURE SCHEMA"
)


schema = {
    "schema":
        "stage26_8d2_cpu_publication_schema_v1",

    "scientific_parent":
        EXPECTED_PARENT,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "stage26_8d1_audit_sha256":
        EXPECTED_8D1_SHA256,

    "cold_start_ci_verdict":
        EXPECTED_8D1_VERDICT,

    "cold_start_publication_policy":
        EXPECTED_8D1_POLICY,

    "publication_label":
        (
            "COMPONENT_LEVEL_WITH_STAGE26_8_"
            "4C3_SENSITIVITY_AUDIT_COMPLETED"
        ),

    "scope": {
        "hardware_phase":
            "CPU_ONLY",

        "gpu_allowed":
            False,

        "publication_artifact_generation_allowed_after_this_lock":
            True,

        "complete_e2e_measurement_available":
            False,

        "cross_group_pareto_allowed":
            False,

        "missing_cost_imputation_allowed":
            False,

        "resource_limit_outcomes_must_be_preserved":
            True,
    },

    "source_anchors":
        SOURCE_ANCHORS,

    "global_rules": {
        "warm_cpu_point_estimates":
            (
                "Use Stage26-2 point estimates. "
                "Preserve PASS, OOM, and timeout exactly."
            ),

        "warm_cpu_uncertainty":
            (
                "Use Stage26-6F1 corrected bootstrap uncertainty only. "
                "Historical Stage26-2 confidence intervals are not certified "
                "and must not appear as publication-facing uncertainty."
            ),

        "cold_start_uncertainty":
            (
                "Point estimates only. Stored Stage26-1 bootstrap confidence "
                "intervals must not appear in publication-facing tables, "
                "figures, captions, or claims."
            ),

        "pareto":
            (
                "Use immutable Stage26-6D descriptive point-estimate "
                "frontiers only. Never recompute and never construct a "
                "cross-group frontier."
            ),

        "capacity_scaling":
            (
                "Use immutable Stage26-7C component-level results only. "
                "No new bootstrap, ratio CI, post-hoc best-batch selection, "
                "or bottleneck claim."
            ),

        "representation":
            (
                "Retain historical Stage26-4C3 as the original measurement. "
                "Report Stage26-8 paired V3/V2 evidence separately as "
                "descriptive implementation sensitivity only. "
                "V3 is not a corrected replacement."
            ),

        "complete_e2e":
            (
                "Report complete E2E as unavailable. "
                "Do not add component latencies, derive pipeline throughput "
                "from minimum component capacity, or impute missing "
                "extraction/bridge costs."
            ),
    },

    # =========================================================================
    # TABLE SCHEMAS
    # =========================================================================

    "tables": [

        # ---------------------------------------------------------------------
        # T1 — Warm isolated inference
        # ---------------------------------------------------------------------

        {
            "id":
                "T26_CPU_WARM_INFERENCE",

            "title":
                "CPU isolated warm-inference deployment profile",

            "row_unit":
                "target x CPU_mode x batch_size",

            "source_policy": [
                "Stage26-2 point estimates",
                "Stage26-6F1 corrected timing uncertainty",
            ],

            "columns": [
                "target_id",
                "group",
                "cpu_mode",
                "batch_size",
                "status",
                "n_timed_observations",

                "p50_latency_ms",
                "p50_ci95_low_ms",
                "p50_ci95_high_ms",

                "p95_latency_ms",
                "p95_ci95_low_ms",
                "p95_ci95_high_ms",

                "p99_latency_ms",
                "p99_ci95_low_ms",
                "p99_ci95_high_ms",

                "median_throughput_samples_per_s",
                "throughput_ci95_low_samples_per_s",
                "throughput_ci95_high_samples_per_s",

                "resource_limit_outcome",
            ],

            "rules": [
                (
                    "For non-PASS/resource-limit rows, preserve unavailable "
                    "metrics as unavailable; never impute."
                ),
                (
                    "p99 and p99 CI are only present when the frozen protocol "
                    "permits them."
                ),
                (
                    "Historical Stage26-2 CI fields are prohibited."
                ),
            ],
        },

        # ---------------------------------------------------------------------
        # T2 — Memory / package
        # ---------------------------------------------------------------------

        {
            "id":
                "T26_CPU_MEMORY_PACKAGE",

            "title":
                "CPU deployment memory and package profile",

            "row_unit":
                "target",

            "source_policy": [
                "Stage26-3B CPU memory/package results"
            ],

            "columns": [
                "target_id",
                "group",
                "deployment_package_size_mib",
                "memory_metric_name",
                "memory_value_mib",
                "memory_status",
                "resource_limit_outcome",
            ],

            "rules": [
                "Preserve frozen RAM-limit OOM outcomes.",
                "Do not infer memory for failed conditions.",
            ],
        },

        # ---------------------------------------------------------------------
        # T3 — Cold start
        # ---------------------------------------------------------------------

        {
            "id":
                "T26_CPU_COLD_START",

            "title":
                "CPU cold-start component profile",

            "row_unit":
                "target x CPU_mode x cold_start_component",

            "source_policy": [
                "Stage26-1 point estimates only"
            ],

            "columns": [
                "target_id",
                "group",
                "cpu_mode",
                "component",
                "n_observations",
                "p50_ms",
                "p95_ms",
                "status",
            ],

            "forbidden_columns": [
                "p50_bootstrap_ci95_low_ms",
                "p50_bootstrap_ci95_high_ms",
                "p95_bootstrap_ci95_low_ms",
                "p95_bootstrap_ci95_high_ms",
            ],

            "rules": [
                "No cold-start CI may be publication-facing.",
                "No post-hoc bootstrap reconstruction.",
            ],
        },

        # ---------------------------------------------------------------------
        # T4 — Capacity / scaling
        # ---------------------------------------------------------------------

        {
            "id":
                "T26_CPU_CAPACITY_SCALING",

            "title":
                "CPU component-level capacity and scaling",

            "row_unit":
                "target x batch_size",

            "source_policy": [
                "Immutable Stage26-7C"
            ],

            "columns": [
                "target_id",
                "group",
                "batch_size",
                "cpu1_status",
                "cpu2_status",
                "within_target_batch_throughput_multiplier",
                "cpu2_over_cpu1_speedup",
                "physical_core_parallel_efficiency",
            ],

            "rules": [
                "No derived-statistic CI.",
                "No post-hoc best-batch selection.",
                (
                    "Do not label any component result as a "
                    "full-pipeline bottleneck."
                ),
            ],
        },

        # ---------------------------------------------------------------------
        # T5 — Component measurements / E2E availability
        # ---------------------------------------------------------------------

        {
            "id":
                "T26_COMPONENT_MEASUREMENTS",

            "title":
                (
                    "Measured deployment components and "
                    "complete-E2E availability"
                ),

            "row_unit":
                "component family",

            "source_policy": [
                "Stage26-4B3 raw extraction component",
                "Stage26-4C3 historical representation component",
                "Stage26-5A E2E availability closure",
            ],

            "columns": [
                "component_family",
                "group_applicability",
                "measurement_status",
                "hardware_mode",
                "batch_size_if_applicable",
                "p50_latency_ms_if_available",
                "p95_latency_ms_if_available",
                "median_throughput_if_available",
                "claim_boundary",
            ],

            "rules": [
                (
                    "Raw extraction is component-only and is not "
                    "CICFlowMeter 70-feature extraction."
                ),
                (
                    "Representation timing uses historical Stage26-4C3."
                ),
                (
                    "Complete E2E must be explicitly marked unavailable."
                ),
                (
                    "No A+B+C additive latency."
                ),
            ],
        },

        # ---------------------------------------------------------------------
        # T6 — Group-B component ratio
        # ---------------------------------------------------------------------

        {
            "id":
                "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO",

            "title":
                "Group-B representation/inference component ratios",

            "row_unit":
                "target x batch_size matched PASS condition",

            "source_policy": [
                "Immutable Stage26-7C"
            ],

            "columns": [
                "target_id",
                "batch_size",
                "representation_over_inference_ratio",
                "publication_label",
            ],

            "rules": [
                (
                    "Use publication label "
                    "COMPONENT_LEVEL_WITH_STAGE26_8_"
                    "4C3_SENSITIVITY_AUDIT_COMPLETED."
                ),
                "No ratio CI.",
                "Not a pipeline bottleneck statement.",
            ],
        },

        # ---------------------------------------------------------------------
        # T7 — Pareto
        # ---------------------------------------------------------------------

        {
            "id":
                "T26_PARETO",

            "title":
                (
                    "Within-group predictive-performance / "
                    "latency Pareto frontier"
                ),

            "row_unit":
                "target",

            "source_policy": [
                "Immutable Stage26-6D"
            ],

            "columns": [
                "group",
                "target_id",
                "pr_auc",
                "cpu1_batch1_p95_latency_ms",
                "pareto_status",
            ],

            "rules": [
                "Group A and Group B are evaluated separately.",
                "No cross-group frontier.",
                "No PR-AUC bootstrap.",
                (
                    "Wording: descriptive point-estimate frontier; "
                    "bootstrap uncertainty reported separately."
                ),
            ],
        },

        # ---------------------------------------------------------------------
        # T8 — Representation sensitivity
        # ---------------------------------------------------------------------

        {
            "id":
                "T26_REPRESENTATION_SENSITIVITY",

            "title":
                (
                    "Paired Stage26-4C3 "
                    "implementation-sensitivity results"
                ),

            "row_unit":
                "batch_size",

            "source_policy": [
                "Stage26-8 paired V2/V3 sensitivity"
            ],

            "columns": [
                "batch_size",
                "n_pairs",
                "fingerprints_equal_count",
                "median_throughput_ratio_v3_over_v2",
                "effect_direction",
                "interpretation",
            ],

            "rules": [
                "Descriptive paired sensitivity only.",
                "No significance test.",
                "No bootstrap.",
                "No materiality threshold.",
                "Do not call V3 a corrected replacement.",
                (
                    "Historical Stage26-4C3 remains "
                    "the original measurement."
                ),
            ],
        },
    ],

    # =========================================================================
    # FIGURE SCHEMAS
    # =========================================================================

    "figures": [

        {
            "id":
                "F26_WARM_LATENCY",

            "title":
                "CPU isolated warm-inference latency by batch size",

            "required_encoding": [
                "separate CPU1 and CPU2 series/panels",
                "p50/p95 point estimates",
                "Stage26-6F1 corrected uncertainty only",
                (
                    "explicit markers/annotations for "
                    "OOM and timeout"
                ),
            ],

            "prohibited": [
                "Stage26-2 historical CIs",
                "interpolation through resource-limit outcomes",
            ],
        },

        {
            "id":
                "F26_WARM_THROUGHPUT",

            "title":
                "CPU isolated warm-inference throughput by batch size",

            "required_encoding": [
                "separate CPU1 and CPU2 series/panels",
                "median throughput",
                "Stage26-6F1 corrected uncertainty only",
                "explicit OOM/timeout preservation",
            ],

            "prohibited": [
                "post-hoc best-batch highlighting"
            ],
        },

        {
            "id":
                "F26_MEMORY_PACKAGE",

            "title":
                "Deployment package size and CPU memory profile",

            "required_encoding": [
                (
                    "package size shown independently "
                    "from runtime memory"
                ),
                "RAM-limit outcomes visibly retained",
            ],

            "prohibited": [
                "imputed memory values"
            ],
        },

        {
            "id":
                "F26_COLD_START",

            "title":
                "CPU cold-start point estimates",

            "required_encoding": [
                "point estimates only"
            ],

            "prohibited": [
                "any cold-start confidence interval"
            ],
        },

        {
            "id":
                "F26_CAPACITY_SCALING",

            "title":
                "CPU component-level scaling",

            "required_encoding": [
                "within-target scaling only",
                (
                    "CPU2/CPU1 speedup where "
                    "matched PASS exists"
                ),
            ],

            "prohibited": [
                "ratio confidence intervals",
                "pipeline bottleneck wording",
                "post-hoc best-batch selection",
            ],
        },

        {
            "id":
                "F26_PARETO",

            "title":
                (
                    "Within-group descriptive "
                    "point-estimate Pareto frontiers"
                ),

            "required_encoding": [
                (
                    "Group A and Group B "
                    "displayed separately"
                ),
                "immutable Stage26-6D membership",
            ],

            "prohibited": [
                "cross-group frontier",
                "PR-AUC bootstrap CI",
                "Pareto recomputation",
            ],
        },

        {
            "id":
                "F26_REPRESENTATION_SENSITIVITY",

            "title":
                "Stage26-4C3 paired implementation sensitivity",

            "required_encoding": [
                "V3/V2 throughput ratio by frozen batch",
                (
                    "neutral reference at ratio 1 "
                    "if plotted"
                ),
                "descriptive wording",
            ],

            "prohibited": [
                "corrected-replacement wording",
                "fixed-bias claim",
                "significance annotation",
            ],
        },

        {
            "id":
                "F26_COMPONENT_BOUNDARY",

            "title":
                "Measured deployment component boundaries",

            "required_encoding": [
                (
                    "raw extraction, representation, and isolated "
                    "inference shown as distinct measured components"
                ),
                (
                    "complete E2E explicitly marked unavailable"
                ),
            ],

            "prohibited": [
                "summed A+B+C latency",
                "complete-pipeline throughput",
                "full-pipeline bottleneck",
            ],
        },
    ],

    # =========================================================================
    # FIXED UNIVERSE
    # =========================================================================

    "fixed_model_groups": {

        "GROUP_A_DUPSAFE70": [
            "XGBoost",
            "LightGBM",
            "CatBoost",
            (
                "FT Transformer final 5-checkpoint "
                "soft-voting ensemble"
            ),
        ],

        "GROUP_B_PACKET_IMAGE": [
            "CNN",
            "ViT",
        ],

        "operational_reference_only": [
            "ENS_LGBM_XGB_EQUAL",
            "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
        ],
    },

    "fixed_cpu_modes": {

        "CPU1": {
            "literal":
                "CPU_1_PHYSICAL_CORE",

            "affinity":
                [0],

            "thread_count":
                1,
        },

        "CPU2": {
            "literal":
                "CPU_2_PHYSICAL_CORE",

            "affinity":
                [0, 1],

            "thread_count":
                2,
        },
    },

    "fixed_batches": [
        1,
        64,
        256,
        1024,
        8192,
    ],

    # =========================================================================
    # WORDING LOCK
    # =========================================================================

    "allowed_wording": [
        "component-level measured capacity",
        "descriptive point-estimate frontier",
        "paired implementation-sensitivity evidence",
        "historical Stage26-4C3 retained",
        "complete E2E measurement unavailable",
        "resource-limit outcome",
        "corrected Stage26-6F1 timing uncertainty",
    ],

    "prohibited_wording": [
        "full-pipeline bottleneck",
        "complete E2E throughput",
        "A+B+C latency",
        "cross-group Pareto frontier",
        "PR-AUC bootstrap CI",
        "V3 corrected replacement for 4C3",
        "historical Stage26-2 CIs as certified",
        "imputed deployment cost",
        "best batch selected after observing results",
    ],

    # =========================================================================
    # POST-LOCK ORDER
    # =========================================================================

    "generation_order_after_lock": [
        "CPU publication tables",
        "CPU publication figures",
        "publication artifact hash/protocol audit",
        "CPU closure commit and push",
        "remote byte verification",
        "clean-repo verification",
        "GPU enablement only after all prior steps pass",
    ],
}


# =============================================================================
# 5. DEFENSIVE SCHEMA ASSERTIONS
# =============================================================================

table_ids = [
    item[
        "id"
    ]
    for item in schema[
        "tables"
    ]
]


figure_ids = [
    item[
        "id"
    ]
    for item in schema[
        "figures"
    ]
]


if len(
    table_ids
) != len(
    set(
        table_ids
    )
):

    raise RuntimeError(
        "Duplicate publication table IDs."
    )


if len(
    figure_ids
) != len(
    set(
        figure_ids
    )
):

    raise RuntimeError(
        "Duplicate publication figure IDs."
    )


if "T26_CPU_COLD_START" not in table_ids:

    raise RuntimeError(
        "Cold-start table schema missing."
    )


if "F26_COLD_START" not in figure_ids:

    raise RuntimeError(
        "Cold-start figure schema missing."
    )


cold_table = next(
    item
    for item in schema[
        "tables"
    ]
    if item[
        "id"
    ]
    ==
    "T26_CPU_COLD_START"
)


for forbidden in [
    "p50_bootstrap_ci95_low_ms",
    "p50_bootstrap_ci95_high_ms",
    "p95_bootstrap_ci95_low_ms",
    "p95_bootstrap_ci95_high_ms",
]:

    if forbidden not in cold_table[
        "forbidden_columns"
    ]:

        raise RuntimeError(
            "Cold-start forbidden field not frozen: "
            f"{forbidden}"
        )


if schema[
    "scope"
][
    "gpu_allowed"
] is not False:

    raise RuntimeError(
        "GPU must remain disabled during CPU publication closure."
    )


# =============================================================================
# 6. WRITE DURABLE SCHEMA LOCK
# =============================================================================

OUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


atomic_json(
    SCHEMA_PATH,
    schema,
)


schema_sha = sha256_file(
    SCHEMA_PATH
)


receipt = {
    "schema":
        "stage26_8d2_cpu_publication_schema_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "measurement_protocol_sha256":
        EXPECTED_PROTOCOL_SHA256,

    "stage26_8d1_audit_sha256":
        EXPECTED_8D1_SHA256,

    "stage26_8d1_verdict":
        EXPECTED_8D1_VERDICT,

    "cold_start_publication_policy":
        EXPECTED_8D1_POLICY,

    "publication_schema_repo_relative_path":
        str(
            SCHEMA_PATH.relative_to(
                REPO
            )
        ),

    "publication_schema_sha256":
        schema_sha,

    "table_ids":
        table_ids,

    "figure_ids":
        figure_ids,

    "table_count":
        len(
            table_ids
        ),

    "figure_count":
        len(
            figure_ids
        ),

    "gpu_used":
        False,

    "timing_executed":
        False,

    "inference_executed":
        False,

    "bootstrap_executed":
        False,

    "pareto_recomputed":
        False,

    "stage26_7c_recomputed":
        False,

    "pcap_accessed":
        False,

    "labels_accessed":
        False,
}


atomic_json(
    RECEIPT_PATH,
    receipt,
)


receipt_sha = sha256_file(
    RECEIPT_PATH
)


manifest = {
    "schema":
        "stage26_8d2_cpu_publication_schema_manifest_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "files": [
        {
            "path":
                str(
                    SCHEMA_PATH.relative_to(
                        REPO
                    )
                ),

            "sha256":
                schema_sha,

            "size_bytes":
                int(
                    SCHEMA_PATH.stat().st_size
                ),
        },

        {
            "path":
                str(
                    RECEIPT_PATH.relative_to(
                        REPO
                    )
                ),

            "sha256":
                receipt_sha,

            "size_bytes":
                int(
                    RECEIPT_PATH.stat().st_size
                ),
        },
    ],
}


atomic_json(
    MANIFEST_PATH,
    manifest,
)


manifest_sha = sha256_file(
    MANIFEST_PATH
)


print(
    "Schema  :",
    SCHEMA_PATH.relative_to(
        REPO
    )
)

print(
    "SHA256  :",
    schema_sha
)

print(
    "Receipt :",
    RECEIPT_PATH.relative_to(
        REPO
    )
)

print(
    "SHA256  :",
    receipt_sha
)

print(
    "Manifest:",
    MANIFEST_PATH.relative_to(
        REPO
    )
)

print(
    "SHA256  :",
    manifest_sha
)

print(
    "Tables  :",
    len(
        table_ids
    ),
    table_ids
)

print(
    "Figures :",
    len(
        figure_ids
    ),
    figure_ids
)

print(
    "Cold-start publication policy:",
    EXPECTED_8D1_POLICY
)

print(
    "GPU allowed:",
    schema[
        "scope"
    ][
        "gpu_allowed"
    ]
)


# =============================================================================
# 7. PRE-COMMIT GIT AUDIT
# =============================================================================

banner(
    "STAGE26-8D2 :: PRE-COMMIT GIT AUDIT"
)


status_lines = [
    line
    for line in git(
        "status",
        "--porcelain",
    ).splitlines()
    if line.strip()
]


for line in status_lines:

    print(
        line
    )


expected_prefix = (
    "?? "
    +
    str(
        OUT_DIR.relative_to(
            REPO
        )
    )
)


if not status_lines:

    raise RuntimeError(
        "No Stage26-8D2 files detected by Git."
    )


if any(
    not line.startswith(
        expected_prefix
    )
    for line in status_lines
):

    raise RuntimeError(
        "Unexpected repository modification detected "
        "before Stage26-8D2 commit."
    )


# =============================================================================
# 8. COMMIT
# =============================================================================

banner(
    "STAGE26-8D2 :: COMMIT SCHEMA FREEZE"
)


git(
    "add",
    str(
        OUT_DIR.relative_to(
            REPO
        )
    ),
)


staged = git(
    "diff",
    "--cached",
    "--name-only",
).splitlines()


expected_staged = sorted(
    [
        str(
            SCHEMA_PATH.relative_to(
                REPO
            )
        ),
        str(
            RECEIPT_PATH.relative_to(
                REPO
            )
        ),
        str(
            MANIFEST_PATH.relative_to(
                REPO
            )
        ),
    ]
)


print(
    "Staged files:"
)


for path in staged:

    print(
        " ",
        path
    )


if sorted(
    staged
) != expected_staged:

    raise RuntimeError(
        "Staged-file set differs from the three "
        "frozen Stage26-8D2 files."
    )


git(
    "commit",
    "-m",
    COMMIT_MESSAGE,
)


new_head = git(
    "rev-parse",
    "HEAD",
)


parent = git(
    "rev-parse",
    "HEAD^",
)


subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "Parent :",
    parent
)

print(
    "HEAD   :",
    new_head
)

print(
    "Subject:",
    subject
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-8D2 commit parent mismatch."
    )


if subject != COMMIT_MESSAGE:

    raise RuntimeError(
        "Unexpected Stage26-8D2 commit subject."
    )


# =============================================================================
# 9. AUTHENTICATED PUSH
# =============================================================================

banner(
    "STAGE26-8D2 :: PUSH"
)


try:

    from kaggle_secrets import UserSecretsClient

except Exception as exc:

    raise RuntimeError(
        "Kaggle UserSecretsClient unavailable."
    ) from exc


try:

    github_token = (
        UserSecretsClient()
        .get_secret(
            "GITHUB_TOKEN"
        )
    )

except Exception as exc:

    raise RuntimeError(
        "Could not read Kaggle Secret GITHUB_TOKEN. "
        "Do not paste the token into notebook source."
    ) from exc


if (
    not github_token
    or
    len(
        github_token.strip()
    )
    <
    20
):

    raise RuntimeError(
        "GITHUB_TOKEN secret is empty or unusable."
    )


askpass_fd, askpass_name = tempfile.mkstemp(
    prefix="stage26_askpass_",
    suffix=".sh",
)


os.close(
    askpass_fd
)


askpass_path = Path(
    askpass_name
)


try:

    askpass_path.write_text(
        "#!/bin/sh\n"
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *Password*) printf "%s\\n" "$STAGE26_GITHUB_TOKEN" ;;\n'
        '  *) printf "%s\\n" "" ;;\n'
        "esac\n",
        encoding="utf-8",
    )


    askpass_path.chmod(
        askpass_path.stat().st_mode
        |
        stat.S_IXUSR
        |
        stat.S_IXGRP
        |
        stat.S_IXOTH
    )


    push_env = os.environ.copy()


    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass_path
    )


    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"


    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token.strip()


    push_output = git(
        "push",
        "origin",
        "main",
        env=push_env,
    )


    print(
        push_output
        if push_output
        else
        "<push completed>"
    )


finally:

    try:

        askpass_path.unlink(
            missing_ok=True
        )

    except Exception:

        pass


    github_token = None


    if "push_env" in locals():

        push_env.pop(
            "STAGE26_GITHUB_TOKEN",
            None,
        )


# =============================================================================
# 10. REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-8D2 :: REMOTE BYTE VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


remote_after = git(
    "rev-parse",
    "origin/main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)


if (
    local_after != new_head
    or
    remote_after != new_head
):

    raise RuntimeError(
        "Local/remote commit mismatch "
        "after Stage26-8D2 push."
    )


verification = []


for path in [
    SCHEMA_PATH,
    RECEIPT_PATH,
    MANIFEST_PATH,
]:

    rel = str(
        path.relative_to(
            REPO
        )
    )


    local_bytes = path.read_bytes()


    remote_bytes = git_blob_bytes(
        "origin/main",
        rel,
    )


    local_sha = sha256_bytes(
        local_bytes
    )


    remote_sha = sha256_bytes(
        remote_bytes
    )


    same = (
        local_bytes
        ==
        remote_bytes
    )


    row = {
        "path":
            rel,

        "local_sha256":
            local_sha,

        "remote_sha256":
            remote_sha,

        "byte_identical":
            same,
    }


    verification.append(
        row
    )


    print(
        f"{'PASS' if same else 'FAIL'} "
        f"{rel}"
    )

    print(
        "  local :",
        local_sha
    )

    print(
        "  remote:",
        remote_sha
    )


    if not same:

        raise RuntimeError(
            "Remote byte verification failed: "
            f"{rel}"
        )


final_status = git(
    "status",
    "--porcelain",
)


# =============================================================================
# 11. FINAL
# =============================================================================

banner(
    "STAGE26-8D2 COMPLETE"
)


print(
    "Durable schema-freeze anchor :",
    new_head
)

print(
    "HEAD == origin/main          :",
    (
        local_after
        ==
        remote_after
        ==
        new_head
    )
)

print(
    "Repo clean                   :",
    final_status == ""
)

print(
    "Remote files byte-identical  :",
    all(
        item[
            "byte_identical"
        ]
        for item in verification
    )
)

print(
    "Tables frozen                :",
    len(
        table_ids
    )
)

print(
    "Figures frozen               :",
    len(
        figure_ids
    )
)

print(
    "Cold-start CIs publication   : PROHIBITED"
)

print(
    "Cold-start point estimates   : ALLOWED"
)

print(
    "Warm timing uncertainty      : STAGE26-6F1 ONLY"
)

print(
    "Historical Stage26-4C3       : RETAINED"
)

print(
    "Stage26-8 sensitivity        : DESCRIPTIVE / SEPARATE"
)

print(
    "Stage26-7C                   : UNCHANGED"
)

print(
    "Stage26-6D Pareto            : UNCHANGED"
)

print(
    "Complete E2E                 : UNAVAILABLE"
)

print(
    "GPU                           : OFF"
)


if final_status:

    raise RuntimeError(
        "Repository is not clean after Stage26-8D2."
    )


print(
    "\nNEXT:"
)

print(
    "  Generate CPU publication tables "
    "from this frozen schema."
)

print(
    "  Do not generate any GPU results yet."
)


STAGE26-8D2 :: DURABLE STATE GATE
Expected parent: fd4497e333ebeb72b1eb188ed64ca2cdd0f08567
Local HEAD     : fd4497e333ebeb72b1eb188ed64ca2cdd0f08567
origin/main    : fd4497e333ebeb72b1eb188ed64ca2cdd0f08567
Repo clean     : True

STAGE26-8D2 :: INPUT IDENTITY
Protocol expected: d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
Protocol actual  : d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
8D1 expected SHA : c90c80936c522df42ba387982bd51b1dfb6548b968a1616ee28d74e7e2800d00
8D1 actual SHA   : c90c80936c522df42ba387982bd51b1dfb6548b968a1616ee28d74e7e2800d00
8D1 verdict       : COLD_START_CI_IMPLEMENTATION_PROVENANCE_NOT_CERTIFIED
Cold-start policy : COLD_START_POINT_ESTIMATES_ONLY

Source anchors:
cold_start                                 46379b6d036008db4d60b056a66f4c01383e3298 commit
warm_cpu_point_estimates                   7ffdba3f4ca4ea5cc53097d62aaf27957009b9f6 commit
cpu_memory_package                         347d93f21d454cc5bda2c45889c890c67c

In [16]:
# =============================================================================
# STAGE26-8D2A
# CPU PUBLICATION SCHEMA SEMANTIC ERRATUM / EFFECTIVE SCHEMA RESOLUTION
#
# SCIENTIFIC PARENT:
#   f04591864f2c64096a3932bc6f793e84a36e81f2
#
# WHY THIS ERRATUM EXISTS
# -----------------------
# Stage26-8D2 correctly froze the publication structure before artifact
# generation. A source-schema inspection performed before table generation
# identified four representation-level ambiguities:
#
#   T26_CPU_MEMORY_PACKAGE
#       - frozen row_unit "target" is insufficient because Stage26-3B memory
#         measurements are prospectively defined per:
#             target x CPU mode x batch
#       - silently aggregating across these conditions would create a new
#         post-hoc statistic.
#
#   T26_CPU_CAPACITY_SCALING
#       - Stage26-7B defines throughput multiplier WITHIN hardware mode.
#       - therefore a singular "within_target_batch_throughput_multiplier"
#         is ambiguous when CPU1 and CPU2 both exist.
#
#   T26_COMPONENT_MEASUREMENTS
#       - raw extraction has multiple distinct measured rates and the original
#         numeric-wide schema would force an arbitrary metric selection.
#       - this table is therefore clarified as a component-boundary /
#         availability matrix. Numeric component details remain in their
#         dedicated frozen tables/results.
#
#   T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO
#       - Stage26-7B prospectively freezes:
#
#           inference throughput / representation throughput
#
#       - the Stage26-7C stored field is:
#
#           component_capacity_ratio_inference_over_representation
#
#       - the Stage26-8D2 publication column label accidentally expressed the
#         reciprocal direction.
#
# IMPORTANT
# ---------
# This erratum:
#
#   - DOES NOT alter Stage26-8D2 historical files
#   - DOES NOT change any measured value
#   - DOES NOT recompute any ratio
#   - DOES NOT invert any ratio
#   - DOES NOT aggregate memory across conditions
#   - DOES NOT inspect performance/result values for selection
#   - reads only protocol/source schemas, headers, and formula locks
#   - creates an effective resolved schema for downstream table generation
#
# NO:
#   timing
#   inference
#   model loading
#   bootstrap
#   random sampling
#   Pareto recomputation
#   Stage26-7C recomputation
#   PCAP
#   labels
#   GPU
#
# DURABLE OUTPUT:
#
#   results/stage26_deployment_profiling/
#       stage26_8d2a_cpu_publication_schema_erratum/
#
#         stage26_8d2a_cpu_publication_schema_erratum.json
#         stage26_8d2a_effective_cpu_publication_schema.json
#         stage26_8d2a_cpu_publication_schema_erratum_receipt.json
#         stage26_8d2a_cpu_publication_schema_erratum_manifest.json
#
# =============================================================================

from __future__ import annotations

import copy
import csv
import hashlib
import json
import os
import stat
import subprocess
import tempfile
from datetime import datetime, timezone
from pathlib import Path


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

RESULT_ROOT = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
)


EXPECTED_PARENT = (
    "f04591864f2c64096a3932bc6f793e84a36e81f2"
)


# -----------------------------------------------------------------------------
# Stage26-8D2 frozen schema
# -----------------------------------------------------------------------------

BASE_SCHEMA_DIR = (
    RESULT_ROOT
    / "stage26_8d2_cpu_publication_schema_lock"
)

BASE_SCHEMA_PATH = (
    BASE_SCHEMA_DIR
    / "stage26_8d2_cpu_publication_schema.json"
)

BASE_RECEIPT_PATH = (
    BASE_SCHEMA_DIR
    / "stage26_8d2_cpu_publication_schema_receipt.json"
)

BASE_MANIFEST_PATH = (
    BASE_SCHEMA_DIR
    / "stage26_8d2_cpu_publication_schema_manifest.json"
)


EXPECTED_BASE_SCHEMA_SHA256 = (
    "f5d66a399ef66f9997de496f109e5ec48c55262b4cd6d7a88e570e8ab7e3cdd6"
)

EXPECTED_BASE_RECEIPT_SHA256 = (
    "63a7b92fef7393457c2540d07cd0b654ef0b22aa718ac44eebb4e8e7d84980e2"
)

EXPECTED_BASE_MANIFEST_SHA256 = (
    "ddc35d05c8e9470f1b90d9eb7f620aafbb359e300418e5bf50e85ce171ac441c"
)


# -----------------------------------------------------------------------------
# Source anchors
# -----------------------------------------------------------------------------

MEMORY_IMPLEMENTATION_ANCHOR = (
    "2e1ece6cc198b054b01e467275951caafb6af794"
)

MEMORY_RESULTS_ANCHOR = (
    "347d93f21d454cc5bda2c45889c890c67cdf0ecc"
)

CAPACITY_FORMULA_ANCHOR = (
    "c524e763b8ae7593204f95f23f9e819b4dcaaf9f"
)

CAPACITY_RESULTS_ANCHOR = (
    "9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3"
)


# -----------------------------------------------------------------------------
# Stage26-3A / 3B
# -----------------------------------------------------------------------------

MEMORY_IMPLEMENTATION = (
    RESULT_ROOT
    / "stage26_3a_memory_implementation_lock"
    / "stage26_3_memory_implementation.json"
)

MEMORY_SUMMARY_CSV = (
    RESULT_ROOT
    / "stage26_3b_cpu_memory_profile"
    / "stage26_3_memory_summary.csv"
)

PACKAGE_SIZE_CSV = (
    RESULT_ROOT
    / "stage26_3b_cpu_memory_profile"
    / "stage26_3_package_sizes.csv"
)


# -----------------------------------------------------------------------------
# Stage26-7B / 7C
# -----------------------------------------------------------------------------

CAPACITY_FORMULA_LOCK = (
    RESULT_ROOT
    / "stage26_7b_capacity_compatibility_lock"
    / "stage26_7b_capacity_formula_lock.json"
)

CAPACITY_BATCH_CSV = (
    RESULT_ROOT
    / "stage26_7c_cpu_capacity_scaling"
    / "stage26_7c_inference_batch_scaling.csv"
)

CAPACITY_TWO_CORE_CSV = (
    RESULT_ROOT
    / "stage26_7c_cpu_capacity_scaling"
    / "stage26_7c_two_core_scaling.csv"
)

GROUP_B_COMPONENT_CSV = (
    RESULT_ROOT
    / "stage26_7c_cpu_capacity_scaling"
    / "stage26_7c_group_b_component_capacity.csv"
)


# -----------------------------------------------------------------------------
# Output
# -----------------------------------------------------------------------------

OUT_DIR = (
    RESULT_ROOT
    / "stage26_8d2a_cpu_publication_schema_erratum"
)

ERRATUM_PATH = (
    OUT_DIR
    / "stage26_8d2a_cpu_publication_schema_erratum.json"
)

EFFECTIVE_SCHEMA_PATH = (
    OUT_DIR
    / "stage26_8d2a_effective_cpu_publication_schema.json"
)

RECEIPT_PATH = (
    OUT_DIR
    / "stage26_8d2a_cpu_publication_schema_erratum_receipt.json"
)

MANIFEST_PATH = (
    OUT_DIR
    / "stage26_8d2a_cpu_publication_schema_erratum_manifest.json"
)


COMMIT_MESSAGE = (
    "stage26: correct CPU publication schema semantics"
)


PUBLICATION_LABEL = (
    "COMPONENT_LEVEL_WITH_STAGE26_8_4C3_SENSITIVITY_AUDIT_COMPLETED"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):

    print(
        "\n"
        + "=" * 124
    )

    print(
        text
    )

    print(
        "=" * 124
    )


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
    env=None,
):

    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
        env=env,
    )

    if (
        check
        and
        p.returncode != 0
    ):

        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(map(str, cmd))}\n"
            f"{p.stdout}"
        )

    return p.stdout.strip()


def git(
    *args,
    check=True,
    env=None,
):

    return run(
        [
            "git",
            *args,
        ],
        check=check,
        env=env,
    )


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                8
                * 1024
                * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def read_json(path):

    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as f:

        return json.load(
            f
        )


def atomic_json(
    path,
    payload,
):

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(path)
        +
        ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def csv_header(path):

    with Path(path).open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:

        reader = csv.reader(
            f
        )

        return next(
            reader
        )


def git_blob_bytes(
    ref,
    rel_path,
):

    p = subprocess.run(
        [
            "git",
            "show",
            f"{ref}:{rel_path}",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            f"Could not read {rel_path} from {ref}:\n"
            +
            p.stderr.decode(
                "utf-8",
                errors="replace",
            )
        )

    return p.stdout


def verify_historical_identity(
    path,
    anchor,
):

    path = Path(
        path
    )

    rel = str(
        path.relative_to(
            REPO
        )
    )

    current = path.read_bytes()

    historical = git_blob_bytes(
        anchor,
        rel,
    )

    current_sha = sha256_bytes(
        current
    )

    historical_sha = sha256_bytes(
        historical
    )

    same = (
        current
        ==
        historical
    )

    print(
        f"{'PASS' if same else 'FAIL'} "
        f"{rel}"
    )

    print(
        "  anchor:",
        anchor
    )

    print(
        "  historical SHA256:",
        historical_sha
    )

    print(
        "  current SHA256   :",
        current_sha
    )

    if not same:

        raise RuntimeError(
            "Source changed since its durable scientific anchor:\n"
            f"{rel}"
        )

    return {
        "path":
            rel,

        "anchor":
            anchor,

        "sha256":
            current_sha,

        "byte_identical_to_anchor":
            True,
    }


def table_by_id(
    schema,
    table_id,
):

    matches = [
        item
        for item in schema[
            "tables"
        ]
        if item[
            "id"
        ]
        ==
        table_id
    ]

    if len(
        matches
    ) != 1:

        raise RuntimeError(
            f"Expected exactly one table schema for {table_id}; "
            f"found {len(matches)}."
        )

    return matches[
        0
    ]


# =============================================================================
# 2. DURABLE STATE GATE
# =============================================================================

banner(
    "STAGE26-8D2A :: DURABLE STATE GATE"
)


git(
    "fetch",
    "origin",
    "main",
)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT
)

print(
    "Local HEAD     :",
    head
)

print(
    "origin/main    :",
    remote
)

print(
    "Repo clean     :",
    status == ""
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected local HEAD before Stage26-8D2A."
    )


if remote != EXPECTED_PARENT:

    raise RuntimeError(
        "origin/main changed before Stage26-8D2A."
    )


if status:

    raise RuntimeError(
        "Repository must be clean before Stage26-8D2A."
    )


if OUT_DIR.exists():

    raise RuntimeError(
        "Stage26-8D2A output directory already exists:\n"
        f"{OUT_DIR}\n\n"
        "Do not overwrite a durable schema erratum."
    )


# =============================================================================
# 3. VERIFY ORIGINAL 8D2 FREEZE
# =============================================================================

banner(
    "STAGE26-8D2A :: VERIFY ORIGINAL 8D2 FREEZE"
)


required_base_files = [
    (
        BASE_SCHEMA_PATH,
        EXPECTED_BASE_SCHEMA_SHA256,
    ),
    (
        BASE_RECEIPT_PATH,
        EXPECTED_BASE_RECEIPT_SHA256,
    ),
    (
        BASE_MANIFEST_PATH,
        EXPECTED_BASE_MANIFEST_SHA256,
    ),
]


for path, expected_sha in required_base_files:

    if not path.is_file():

        raise FileNotFoundError(
            path
        )

    actual_sha = sha256_file(
        path
    )

    print(
        path.name
    )

    print(
        "  expected:",
        expected_sha
    )

    print(
        "  actual  :",
        actual_sha
    )

    if actual_sha != expected_sha:

        raise RuntimeError(
            "Stage26-8D2 frozen file identity mismatch:\n"
            f"{path}"
        )


base_schema = read_json(
    BASE_SCHEMA_PATH
)


base_table_ids = [
    item[
        "id"
    ]
    for item in base_schema[
        "tables"
    ]
]


if len(
    base_table_ids
) != 8:

    raise RuntimeError(
        "Expected exactly eight Stage26-8D2 tables."
    )


if len(
    base_schema[
        "figures"
    ]
) != 8:

    raise RuntimeError(
        "Expected exactly eight Stage26-8D2 figures."
    )


print(
    "\nOriginal schema table count:",
    len(
        base_table_ids
    )
)

print(
    "Original schema figure count:",
    len(
        base_schema[
            "figures"
        ]
    )
)


# =============================================================================
# 4. VERIFY SOURCE FILES + HISTORICAL BYTE IDENTITY
# =============================================================================

banner(
    "STAGE26-8D2A :: SOURCE BYTE IDENTITY"
)


required_source_files = [
    MEMORY_IMPLEMENTATION,
    MEMORY_SUMMARY_CSV,
    PACKAGE_SIZE_CSV,
    CAPACITY_FORMULA_LOCK,
    CAPACITY_BATCH_CSV,
    CAPACITY_TWO_CORE_CSV,
    GROUP_B_COMPONENT_CSV,
]


for path in required_source_files:

    if not path.is_file():

        raise FileNotFoundError(
            path
        )


source_identity = []


source_identity.append(
    verify_historical_identity(
        MEMORY_IMPLEMENTATION,
        MEMORY_IMPLEMENTATION_ANCHOR,
    )
)


source_identity.append(
    verify_historical_identity(
        MEMORY_SUMMARY_CSV,
        MEMORY_RESULTS_ANCHOR,
    )
)


source_identity.append(
    verify_historical_identity(
        PACKAGE_SIZE_CSV,
        MEMORY_RESULTS_ANCHOR,
    )
)


source_identity.append(
    verify_historical_identity(
        CAPACITY_FORMULA_LOCK,
        CAPACITY_FORMULA_ANCHOR,
    )
)


source_identity.append(
    verify_historical_identity(
        CAPACITY_BATCH_CSV,
        CAPACITY_RESULTS_ANCHOR,
    )
)


source_identity.append(
    verify_historical_identity(
        CAPACITY_TWO_CORE_CSV,
        CAPACITY_RESULTS_ANCHOR,
    )
)


source_identity.append(
    verify_historical_identity(
        GROUP_B_COMPONENT_CSV,
        CAPACITY_RESULTS_ANCHOR,
    )
)


# =============================================================================
# 5. SOURCE-SCHEMA-ONLY AUDIT
#
# IMPORTANT:
#   No CSV result rows are read below.
#   Only headers and prospective implementation/formula metadata are read.
# =============================================================================

banner(
    "STAGE26-8D2A :: SOURCE-SCHEMA-ONLY AUDIT"
)


memory_impl = read_json(
    MEMORY_IMPLEMENTATION
)

capacity_formula = read_json(
    CAPACITY_FORMULA_LOCK
)


# -----------------------------------------------------------------------------
# Memory prospective geometry
# -----------------------------------------------------------------------------

memory_scope = memory_impl[
    "memory_scope"
]

memory_boundaries = memory_impl[
    "memory_boundaries"
]


print(
    "Memory target count:",
    memory_scope[
        "target_count"
    ]
)

print(
    "Memory CPU modes:",
    memory_scope[
        "cpu_modes"
    ]
)

print(
    "Memory batches:",
    memory_scope[
        "batch_sizes"
    ]
)

print(
    "Memory source conditions:",
    memory_scope[
        "source_condition_count"
    ]
)


if memory_scope[
    "target_count"
] != 8:

    raise RuntimeError(
        "Unexpected Stage26-3 target count."
    )


if memory_scope[
    "cpu_modes"
] != 2:

    raise RuntimeError(
        "Stage26-3 memory does not contain two CPU modes."
    )


if memory_scope[
    "batch_sizes"
] != [
    1,
    64,
    256,
    1024,
    8192,
]:

    raise RuntimeError(
        "Stage26-3 frozen memory batches changed."
    )


if memory_scope[
    "source_condition_count"
] != 80:

    raise RuntimeError(
        "Stage26-3 frozen memory condition count changed."
    )


required_memory_boundaries = {
    "baseline_rss",
    "loaded_rss",
    "peak_rss",
    "delta_model_rss",
    "delta_peak_rss",
    "ru_maxrss",
}


if not required_memory_boundaries.issubset(
    set(
        memory_boundaries
    )
):

    raise RuntimeError(
        "Frozen Stage26-3 memory boundary set changed."
    )


print(
    "\nFrozen memory boundaries:"
)


for key in sorted(
    memory_boundaries
):

    print(
        " ",
        key,
        "->",
        memory_boundaries[
            key
        ]
    )


# -----------------------------------------------------------------------------
# CSV headers only
# -----------------------------------------------------------------------------

memory_header = csv_header(
    MEMORY_SUMMARY_CSV
)

package_header = csv_header(
    PACKAGE_SIZE_CSV
)

batch_header = csv_header(
    CAPACITY_BATCH_CSV
)

two_core_header = csv_header(
    CAPACITY_TWO_CORE_CSV
)

group_b_header = csv_header(
    GROUP_B_COMPONENT_CSV
)


required_memory_columns = {
    "target_id",
    "comparison_group",
    "hardware_mode",
    "batch_size",
    "planned_repetitions",
    "pass_repetitions",
    "resource_limit_oom_repetitions",
    "timeout_resource_limit_repetitions",
    "median_baseline_rss_bytes",
    "median_loaded_rss_bytes",
    "median_peak_rss_bytes",
    "median_delta_model_rss_bytes",
    "median_delta_peak_rss_bytes",
    "median_ru_maxrss_bytes_descriptive",
}


if not required_memory_columns.issubset(
    set(
        memory_header
    )
):

    missing = sorted(
        required_memory_columns
        -
        set(
            memory_header
        )
    )

    raise RuntimeError(
        "Required Stage26-3B memory columns missing:\n"
        +
        "\n".join(
            missing
        )
    )


required_package_columns = {
    "target_id",
    "deployment_package_size_mib",
}


if not required_package_columns.issubset(
    set(
        package_header
    )
):

    raise RuntimeError(
        "Stage26-3B package-size schema changed."
    )


required_batch_columns = {
    "target_id",
    "hardware_mode",
    "batch_size",
    "status",
    "throughput_multiplier_vs_B1",
    "claim_boundary",
    "uncertainty_propagated",
}


if not required_batch_columns.issubset(
    set(
        batch_header
    )
):

    raise RuntimeError(
        "Stage26-7C batch-scaling schema changed."
    )


required_two_core_columns = {
    "target_id",
    "batch_size",
    "two_core_speedup",
    "two_core_parallel_efficiency",
    "claim_boundary",
    "uncertainty_propagated",
}


if not required_two_core_columns.issubset(
    set(
        two_core_header
    )
):

    raise RuntimeError(
        "Stage26-7C two-core scaling schema changed."
    )


required_group_b_columns = {
    "target_id",
    "hardware_mode",
    "batch_size",
    "component_capacity_ratio_inference_over_representation",
    "lower_measured_capacity_component",
    "claim_boundary",
    "uncertainty_propagated",
}


if not required_group_b_columns.issubset(
    set(
        group_b_header
    )
):

    raise RuntimeError(
        "Stage26-7C Group-B capacity schema changed."
    )


print(
    "\nMemory header structural gate: PASS"
)

print(
    "Package header structural gate: PASS"
)

print(
    "Batch-scaling header structural gate: PASS"
)

print(
    "Two-core header structural gate: PASS"
)

print(
    "Group-B ratio header structural gate: PASS"
)


# =============================================================================
# 6. PROSPECTIVE CAPACITY FORMULA GATES
# =============================================================================

banner(
    "STAGE26-8D2A :: PROSPECTIVE FORMULA GATES"
)


batch_scope = (
    capacity_formula[
        "batch_scaling"
    ][
        "scope"
    ]
)

batch_formula = (
    capacity_formula[
        "batch_scaling"
    ][
        "throughput_multiplier"
    ][
        "formula"
    ]
)

ratio_formula = (
    capacity_formula[
        "group_B_representation_inference_capacity"
    ][
        "ratio"
    ][
        "formula"
    ]
)

ratio_name = (
    capacity_formula[
        "group_B_representation_inference_capacity"
    ][
        "ratio"
    ][
        "name"
    ]
)


print(
    "Batch scaling scope:"
)

print(
    " ",
    batch_scope
)

print(
    "Batch throughput formula:"
)

print(
    " ",
    batch_formula
)

print(
    "Group-B ratio name:"
)

print(
    " ",
    ratio_name
)

print(
    "Group-B ratio formula:"
)

print(
    " ",
    ratio_formula
)


if batch_scope != "WITHIN_TARGET_WITHIN_HARDWARE_MODE":

    raise RuntimeError(
        "Frozen Stage26-7B batch-scaling scope changed."
    )


if batch_formula != (
    "T(target,hardware,batch) / T(target,hardware,1)"
):

    raise RuntimeError(
        "Frozen Stage26-7B throughput-multiplier formula changed."
    )


if ratio_formula != (
    "inference_median_throughput_flows_per_second / "
    "representation_median_flows_per_second"
):

    raise RuntimeError(
        "Frozen Stage26-7B Group-B ratio formula changed."
    )


if ratio_name != "component_capacity_ratio":

    raise RuntimeError(
        "Frozen Stage26-7B ratio name changed."
    )


# =============================================================================
# 7. CONFIRM EXACT 8D2 AMBIGUITIES BEFORE WRITING ERRATUM
# =============================================================================

banner(
    "STAGE26-8D2A :: CONFIRM BASE-SCHEMA AMBIGUITIES"
)


base_t2 = table_by_id(
    base_schema,
    "T26_CPU_MEMORY_PACKAGE",
)

base_t4 = table_by_id(
    base_schema,
    "T26_CPU_CAPACITY_SCALING",
)

base_t5 = table_by_id(
    base_schema,
    "T26_COMPONENT_MEASUREMENTS",
)

base_t6 = table_by_id(
    base_schema,
    "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO",
)


if base_t2[
    "row_unit"
] != "target":

    raise RuntimeError(
        "T26_CPU_MEMORY_PACKAGE no longer has the "
        "specific row-unit ambiguity being corrected."
    )


if (
    "within_target_batch_throughput_multiplier"
    not in
    base_t4[
        "columns"
    ]
):

    raise RuntimeError(
        "T26_CPU_CAPACITY_SCALING no longer has the "
        "specific singular multiplier ambiguity being corrected."
    )


if (
    "representation_over_inference_ratio"
    not in
    base_t6[
        "columns"
    ]
):

    raise RuntimeError(
        "T26 Group-B ratio no longer has the reciprocal label "
        "being corrected."
    )


old_t5_numeric_columns = {
    "p50_latency_ms_if_available",
    "p95_latency_ms_if_available",
    "median_throughput_if_available",
}


if not old_t5_numeric_columns.issubset(
    set(
        base_t5[
            "columns"
        ]
    )
):

    raise RuntimeError(
        "T26_COMPONENT_MEASUREMENTS no longer has the "
        "numeric-wide schema being clarified."
    )


print(
    "T2 memory row-unit ambiguity         : CONFIRMED"
)

print(
    "T4 hardware-specific multiplier issue: CONFIRMED"
)

print(
    "T5 component metric selection issue  : CONFIRMED"
)

print(
    "T6 reciprocal ratio-label issue       : CONFIRMED"
)


# =============================================================================
# 8. FREEZE ERRATUM OVERRIDES
# =============================================================================

banner(
    "STAGE26-8D2A :: FREEZE SEMANTIC OVERRIDES"
)


T2_OVERRIDE = {
    "id":
        "T26_CPU_MEMORY_PACKAGE",

    "title":
        "CPU deployment memory and package profile",

    "row_unit":
        (
            "target x CPU_mode x batch_size x memory_metric_name"
        ),

    "source_policy": [
        "Stage26-3B condition summaries",
        "Stage26-3B deployment package sizes",
    ],

    "columns": [
        "target_id",
        "group",
        "cpu_mode",
        "batch_size",
        "deployment_package_size_mib",
        "memory_metric_name",
        "memory_value_mib",
        "planned_repetitions",
        "pass_repetitions",
        "resource_limit_oom_repetitions",
        "timeout_resource_limit_repetitions",
        "memory_status",
        "resource_limit_outcome",
    ],

    "frozen_primary_memory_metrics": [
        {
            "output_name":
                "median_baseline_rss_mib",

            "source_field":
                "median_baseline_rss_bytes",
        },
        {
            "output_name":
                "median_loaded_rss_mib",

            "source_field":
                "median_loaded_rss_bytes",
        },
        {
            "output_name":
                "median_peak_rss_mib",

            "source_field":
                "median_peak_rss_bytes",
        },
        {
            "output_name":
                "median_delta_model_rss_mib",

            "source_field":
                "median_delta_model_rss_bytes",
        },
        {
            "output_name":
                "median_delta_peak_rss_mib",

            "source_field":
                "median_delta_peak_rss_bytes",
        },
    ],

    "descriptive_metric_excluded_from_primary_table": {
        "source_field":
            "median_ru_maxrss_bytes_descriptive",

        "reason":
            (
                "Stage26-3A prospectively labels ru_maxrss "
                "as descriptive only."
            ),
    },

    "unit_conversion": {
        "from":
            "bytes",

        "to":
            "MiB",

        "formula":
            "bytes / 1048576",

        "classification":
            "UNIT_CONVERSION_ONLY_NOT_NEW_STATISTIC",
    },

    "rules": [
        (
            "Do not aggregate across CPU modes or batch sizes."
        ),
        (
            "Do not select a best, worst, maximum, or minimum "
            "condition after observing results."
        ),
        (
            "Every frozen Stage26-3B condition remains represented."
        ),
        (
            "Deployment package size is joined by target and may be "
            "repeated across condition/metric rows."
        ),
        (
            "Preserve planned/pass/OOM/timeout repetition counts."
        ),
        (
            "If a memory value is unavailable because of a frozen "
            "resource-limit outcome, retain it as unavailable; "
            "never impute."
        ),
    ],
}


T4_OVERRIDE = {
    "id":
        "T26_CPU_CAPACITY_SCALING",

    "title":
        "CPU component-level capacity and scaling",

    "row_unit":
        "target x batch_size",

    "source_policy": [
        "Immutable Stage26-7C inference batch scaling",
        "Immutable Stage26-7C two-core scaling",
    ],

    "columns": [
        "target_id",
        "group",
        "batch_size",
        "cpu1_status",
        "cpu2_status",
        "cpu1_throughput_multiplier_vs_b1",
        "cpu2_throughput_multiplier_vs_b1",
        "cpu2_over_cpu1_speedup",
        "physical_core_parallel_efficiency",
    ],

    "source_field_mapping": {
        "cpu1_throughput_multiplier_vs_b1":
            (
                "stage26_7c_inference_batch_scaling.csv::"
                "throughput_multiplier_vs_B1 where "
                "hardware_mode=CPU_1_PHYSICAL_CORE"
            ),

        "cpu2_throughput_multiplier_vs_b1":
            (
                "stage26_7c_inference_batch_scaling.csv::"
                "throughput_multiplier_vs_B1 where "
                "hardware_mode=CPU_2_PHYSICAL_CORE"
            ),

        "cpu2_over_cpu1_speedup":
            (
                "stage26_7c_two_core_scaling.csv::"
                "two_core_speedup"
            ),

        "physical_core_parallel_efficiency":
            (
                "stage26_7c_two_core_scaling.csv::"
                "two_core_parallel_efficiency"
            ),
    },

    "rules": [
        (
            "Copy immutable Stage26-7C descriptive values; "
            "do not recompute them."
        ),
        (
            "Batch throughput multiplier remains within target "
            "and within hardware mode exactly as Stage26-7B froze."
        ),
        (
            "Two-core speedup/efficiency appear only where "
            "Stage26-7C contains the matched PASS result."
        ),
        "No derived-statistic CI.",
        "No post-hoc best-batch selection.",
        (
            "Do not label any component result as a "
            "full-pipeline bottleneck."
        ),
    ],
}


T5_OVERRIDE = {
    "id":
        "T26_COMPONENT_MEASUREMENTS",

    "title":
        (
            "Measured deployment component boundaries and "
            "complete-E2E availability"
        ),

    "row_unit":
        "component family",

    "source_policy": [
        "Stage26-4B3 raw extraction component",
        "Stage26-4C3 historical representation component",
        (
            "Stage26-2 point estimates + Stage26-6F1 corrected "
            "uncertainty for isolated inference"
        ),
        "Stage26-5A complete-E2E availability closure",
    ],

    "columns": [
        "component_family",
        "group_applicability",
        "measurement_status",
        "source_stage",
        "hardware_scope",
        "quantitative_detail_reference",
        "claim_boundary",
        "complete_e2e_available",
    ],

    "fixed_rows": [
        {
            "component_family":
                "RAW_EXTRACTION_COMPONENT",

            "group_applicability":
                "RAW_PACKET_EXTRACTION_COMPONENT_ONLY",

            "measurement_status":
                "MEASURED_COMPONENT_ONLY",

            "source_stage":
                "Stage26-4B3",

            "hardware_scope":
                "CPU_1_PHYSICAL_CORE",

            "quantitative_detail_reference":
                "Stage26-4B3 committed extraction summary",

            "claim_boundary":
                (
                    "Not CICFlowMeter 70-feature extraction; "
                    "not complete E2E; no full-pipeline extrapolation."
                ),

            "complete_e2e_available":
                False,
        },
        {
            "component_family":
                "PACKET_IMAGE_REPRESENTATION_COMPONENT",

            "group_applicability":
                "GROUP_B_PACKET_IMAGE",

            "measurement_status":
                "MEASURED_COMPONENT_ONLY",

            "source_stage":
                "Historical Stage26-4C3 retained",

            "hardware_scope":
                "CPU_1_PHYSICAL_CORE",

            "quantitative_detail_reference":
                (
                    "Stage26-4C3 historical representation summary; "
                    "Stage26-8 sensitivity reported separately"
                ),

            "claim_boundary":
                (
                    "Historical Stage26-4C3 retained as original "
                    "measurement; V3 is sensitivity only."
                ),

            "complete_e2e_available":
                False,
        },
        {
            "component_family":
                "ISOLATED_WARM_INFERENCE_COMPONENT",

            "group_applicability":
                "GROUP_A_DUPSAFE70_AND_GROUP_B_PACKET_IMAGE",

            "measurement_status":
                "MEASURED_COMPONENT_ONLY",

            "source_stage":
                "Stage26-2 + Stage26-6F1",

            "hardware_scope":
                "CPU1_AND_CPU2",

            "quantitative_detail_reference":
                "T26_CPU_WARM_INFERENCE",

            "claim_boundary":
                (
                    "Isolated warm inference only; corrected "
                    "Stage26-6F1 timing uncertainty."
                ),

            "complete_e2e_available":
                False,
        },
        {
            "component_family":
                "COMPLETE_E2E",

            "group_applicability":
                "GROUP_A_DUPSAFE70_AND_GROUP_B_PACKET_IMAGE",

            "measurement_status":
                "CLOSED_NO_VALID_COMPLETE_E2E_MEASUREMENT",

            "source_stage":
                "Stage26-5A",

            "hardware_scope":
                "NOT_APPLICABLE",

            "quantitative_detail_reference":
                "NONE",

            "claim_boundary":
                (
                    "Complete E2E measurement unavailable; "
                    "no A+B+C additive latency; "
                    "no throughput from minimum component capacity; "
                    "no missing-cost imputation."
                ),

            "complete_e2e_available":
                False,
        },
    ],

    "rules": [
        (
            "This table is a boundary/availability matrix, not a "
            "new numerical synthesis."
        ),
        (
            "Numeric component measurements remain in their "
            "original committed sources or dedicated publication tables."
        ),
        (
            "Do not choose one raw-extraction rate post hoc merely "
            "to populate a generic throughput column."
        ),
        (
            "Do not combine raw extraction, representation, and "
            "inference into a complete pipeline estimate."
        ),
    ],
}


T6_OVERRIDE = {
    "id":
        "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO",

    "title":
        (
            "Group-B inference-to-representation "
            "component-capacity ratio"
        ),

    "row_unit":
        "target x batch_size matched PASS condition",

    "source_policy": [
        "Immutable Stage26-7C"
    ],

    "columns": [
        "target_id",
        "batch_size",
        "inference_over_representation_capacity_ratio",
        "publication_label",
    ],

    "source_field_mapping": {
        "inference_over_representation_capacity_ratio":
            (
                "stage26_7c_group_b_component_capacity.csv::"
                "component_capacity_ratio_inference_over_representation"
            ),
    },

    "frozen_formula":
        (
            "inference_median_throughput_flows_per_second / "
            "representation_median_flows_per_second"
        ),

    "rules": [
        (
            "Copy the existing immutable Stage26-7C ratio directly."
        ),
        (
            "Do not invert, reciprocate, or recompute the ratio."
        ),
        (
            "Use publication label "
            "COMPONENT_LEVEL_WITH_STAGE26_8_"
            "4C3_SENSITIVITY_AUDIT_COMPLETED."
        ),
        "No ratio CI.",
        (
            "The ratio is a downstream component-capacity "
            "comparison, not a pipeline bottleneck statement."
        ),
    ],
}


TABLE_OVERRIDES = {
    "T26_CPU_MEMORY_PACKAGE":
        T2_OVERRIDE,

    "T26_CPU_CAPACITY_SCALING":
        T4_OVERRIDE,

    "T26_COMPONENT_MEASUREMENTS":
        T5_OVERRIDE,

    "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO":
        T6_OVERRIDE,
}


# =============================================================================
# 9. BUILD EFFECTIVE RESOLVED SCHEMA
# =============================================================================

effective_schema = copy.deepcopy(
    base_schema
)


effective_schema[
    "schema"
] = (
    "stage26_8d2a_effective_cpu_publication_schema_v1"
)


effective_schema[
    "schema_resolution"
] = {
    "base_schema_commit":
        EXPECTED_PARENT,

    "base_schema_sha256":
        EXPECTED_BASE_SCHEMA_SHA256,

    "resolution":
        (
            "STAGE26_8D2_BASE_PLUS_STAGE26_8D2A_"
            "SEMANTIC_ERRATUM"
        ),

    "original_stage26_8d2_files_modified":
        False,

    "result_values_changed":
        False,

    "statistics_recomputed":
        False,

    "ratios_recomputed":
        False,

    "ratio_reciprocals_computed":
        False,

    "memory_conditions_aggregated":
        False,

    "effective_for_downstream_publication_generation":
        True,
}


new_tables = []


for table in effective_schema[
    "tables"
]:

    table_id = table[
        "id"
    ]

    if table_id in TABLE_OVERRIDES:

        new_tables.append(
            copy.deepcopy(
                TABLE_OVERRIDES[
                    table_id
                ]
            )
        )

    else:

        new_tables.append(
            table
        )


effective_schema[
    "tables"
] = new_tables


effective_schema[
    "publication_label"
] = PUBLICATION_LABEL


effective_schema[
    "global_rules"
][
    "schema_erratum"
] = (
    "Stage26-8D2A resolves table-shape/semantic ambiguities only. "
    "No scientific result or protocol conclusion changes."
)


# =============================================================================
# 10. DEFENSIVE EFFECTIVE-SCHEMA ASSERTIONS
# =============================================================================

effective_ids = [
    item[
        "id"
    ]
    for item in effective_schema[
        "tables"
    ]
]


if effective_ids != base_table_ids:

    raise RuntimeError(
        "Stage26-8D2A changed table identity/order."
    )


if len(
    effective_schema[
        "figures"
    ]
) != 8:

    raise RuntimeError(
        "Stage26-8D2A changed figure count."
    )


# T2 must now carry exact condition geometry.
effective_t2 = table_by_id(
    effective_schema,
    "T26_CPU_MEMORY_PACKAGE",
)


for required in [
    "cpu_mode",
    "batch_size",
    "memory_metric_name",
    "memory_value_mib",
]:

    if required not in effective_t2[
        "columns"
    ]:

        raise RuntimeError(
            f"Effective T2 missing {required}."
        )


# T4 must have separate CPU1/CPU2 batch multipliers.
effective_t4 = table_by_id(
    effective_schema,
    "T26_CPU_CAPACITY_SCALING",
)


if (
    "within_target_batch_throughput_multiplier"
    in
    effective_t4[
        "columns"
    ]
):

    raise RuntimeError(
        "Ambiguous singular T4 multiplier survived erratum."
    )


for required in [
    "cpu1_throughput_multiplier_vs_b1",
    "cpu2_throughput_multiplier_vs_b1",
]:

    if required not in effective_t4[
        "columns"
    ]:

        raise RuntimeError(
            f"Effective T4 missing {required}."
        )


# T5 must no longer force arbitrary numeric component selection.
effective_t5 = table_by_id(
    effective_schema,
    "T26_COMPONENT_MEASUREMENTS",
)


for old_numeric in old_t5_numeric_columns:

    if old_numeric in effective_t5[
        "columns"
    ]:

        raise RuntimeError(
            "Ambiguous numeric-wide T5 field survived erratum: "
            f"{old_numeric}"
        )


if len(
    effective_t5[
        "fixed_rows"
    ]
) != 4:

    raise RuntimeError(
        "Effective T5 must contain four frozen component families."
    )


# T6 must use prospective formula direction.
effective_t6 = table_by_id(
    effective_schema,
    "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO",
)


if (
    "representation_over_inference_ratio"
    in
    effective_t6[
        "columns"
    ]
):

    raise RuntimeError(
        "Reciprocal T6 label survived erratum."
    )


if (
    "inference_over_representation_capacity_ratio"
    not in
    effective_t6[
        "columns"
    ]
):

    raise RuntimeError(
        "Correct Stage26-7B ratio direction missing from effective T6."
    )


if effective_t6[
    "frozen_formula"
] != ratio_formula:

    raise RuntimeError(
        "Effective T6 formula differs from prospective Stage26-7B."
    )


print(
    "Effective table IDs preserved:",
    len(
        effective_ids
    )
)

print(
    "Effective figure count preserved:",
    len(
        effective_schema[
            "figures"
        ]
    )
)

print(
    "T2 memory condition geometry : RESOLVED"
)

print(
    "T4 CPU1/CPU2 multipliers     : RESOLVED"
)

print(
    "T5 component boundary matrix : RESOLVED"
)

print(
    "T6 ratio direction           : RESOLVED"
)


# =============================================================================
# 11. WRITE ERRATUM + EFFECTIVE SCHEMA
# =============================================================================

banner(
    "STAGE26-8D2A :: WRITE DURABLE ERRATUM"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


erratum_payload = {
    "schema":
        "stage26_8d2a_cpu_publication_schema_semantic_erratum_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "base_schema": {
        "repo_relative_path":
            str(
                BASE_SCHEMA_PATH.relative_to(
                    REPO
                )
            ),

        "sha256":
            EXPECTED_BASE_SCHEMA_SHA256,

        "status":
            "RETAINED_UNMODIFIED_AS_HISTORICAL_FREEZE",
    },

    "trigger":
        (
            "SOURCE_SCHEMA_SEMANTIC_AMBIGUITIES_FOUND_BEFORE_"
            "PUBLICATION_TABLE_GENERATION"
        ),

    "audit_scope":
        (
            "PROTOCOL_FORMULA_AND_SOURCE_HEADER_STRUCTURE_ONLY"
        ),

    "result_rows_used_for_schema_selection":
        False,

    "scientific_results_changed":
        False,

    "statistics_recomputed":
        False,

    "ratio_recomputed":
        False,

    "ratio_inverted":
        False,

    "memory_aggregated_across_conditions":
        False,

    "corrections": [
        {
            "table_id":
                "T26_CPU_MEMORY_PACKAGE",

            "issue":
                (
                    "Base row unit 'target' omitted prospectively frozen "
                    "CPU-mode and batch condition dimensions."
                ),

            "resolution":
                (
                    "Use target x CPU_mode x batch_size x "
                    "memory_metric_name long form; no condition aggregation."
                ),
        },
        {
            "table_id":
                "T26_CPU_CAPACITY_SCALING",

            "issue":
                (
                    "Base schema used one singular batch throughput "
                    "multiplier although Stage26-7B freezes scaling "
                    "within hardware mode."
                ),

            "resolution":
                (
                    "Expose separate CPU1 and CPU2 immutable Stage26-7C "
                    "throughput multipliers."
                ),
        },
        {
            "table_id":
                "T26_COMPONENT_MEASUREMENTS",

            "issue":
                (
                    "Base numeric-wide component schema could require "
                    "arbitrary selection among multiple valid raw-extraction "
                    "component metrics."
                ),

            "resolution":
                (
                    "Use a component-boundary/availability matrix; "
                    "numeric details remain in dedicated committed sources "
                    "and publication tables."
                ),
        },
        {
            "table_id":
                "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO",

            "issue":
                (
                    "Base output column label expressed the reciprocal "
                    "direction of the prospective Stage26-7B formula."
                ),

            "resolution":
                (
                    "Label the existing immutable ratio as "
                    "inference_over_representation_capacity_ratio; "
                    "no value inversion or recomputation."
                ),
        },
    ],

    "prospective_formula_evidence": {
        "batch_scaling_scope":
            batch_scope,

        "batch_throughput_formula":
            batch_formula,

        "group_b_ratio_formula":
            ratio_formula,

        "group_b_ratio_name":
            ratio_name,
    },

    "source_identity":
        source_identity,

    "table_overrides":
        TABLE_OVERRIDES,

    "publication_policy": {
        "effective_schema":
            (
                "STAGE26_8D2_BASE_PLUS_STAGE26_8D2A_ERRATUM"
            ),

        "cold_start":
            "COLD_START_POINT_ESTIMATES_ONLY",

        "warm_timing_uncertainty":
            "STAGE26_6F1_CORRECTED_ONLY",

        "historical_4c3":
            "RETAIN_AS_ORIGINAL_MEASUREMENT",

        "stage26_8_representation_sensitivity":
            "DESCRIPTIVE_SEPARATE_NOT_REPLACEMENT",

        "stage26_7c":
            "UNCHANGED",

        "stage26_6d_pareto":
            "UNCHANGED",

        "complete_e2e":
            "UNAVAILABLE",

        "gpu":
            "OFF",
    },
}


atomic_json(
    ERRATUM_PATH,
    erratum_payload,
)


atomic_json(
    EFFECTIVE_SCHEMA_PATH,
    effective_schema,
)


erratum_sha = sha256_file(
    ERRATUM_PATH
)

effective_schema_sha = sha256_file(
    EFFECTIVE_SCHEMA_PATH
)


# =============================================================================
# 12. WRITE RECEIPT
# =============================================================================

receipt_payload = {
    "schema":
        "stage26_8d2a_cpu_publication_schema_erratum_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "base_schema_sha256":
        EXPECTED_BASE_SCHEMA_SHA256,

    "erratum_repo_relative_path":
        str(
            ERRATUM_PATH.relative_to(
                REPO
            )
        ),

    "erratum_sha256":
        erratum_sha,

    "effective_schema_repo_relative_path":
        str(
            EFFECTIVE_SCHEMA_PATH.relative_to(
                REPO
            )
        ),

    "effective_schema_sha256":
        effective_schema_sha,

    "corrected_table_ids": [
        "T26_CPU_MEMORY_PACKAGE",
        "T26_CPU_CAPACITY_SCALING",
        "T26_COMPONENT_MEASUREMENTS",
        "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO",
    ],

    "table_count":
        len(
            effective_schema[
                "tables"
            ]
        ),

    "figure_count":
        len(
            effective_schema[
                "figures"
            ]
        ),

    "result_rows_used_for_schema_selection":
        False,

    "timing_executed":
        False,

    "inference_executed":
        False,

    "model_loaded":
        False,

    "bootstrap_executed":
        False,

    "random_sampling_executed":
        False,

    "ratio_recomputed":
        False,

    "ratio_inverted":
        False,

    "memory_condition_aggregation":
        False,

    "pareto_recomputed":
        False,

    "stage26_7c_recomputed":
        False,

    "pcap_accessed":
        False,

    "labels_accessed":
        False,

    "gpu_used":
        False,
}


atomic_json(
    RECEIPT_PATH,
    receipt_payload,
)


receipt_sha = sha256_file(
    RECEIPT_PATH
)


# =============================================================================
# 13. WRITE MANIFEST
# =============================================================================

manifest_payload = {
    "schema":
        "stage26_8d2a_cpu_publication_schema_erratum_manifest_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "files": [
        {
            "path":
                str(
                    ERRATUM_PATH.relative_to(
                        REPO
                    )
                ),

            "sha256":
                erratum_sha,

            "size_bytes":
                int(
                    ERRATUM_PATH.stat().st_size
                ),
        },
        {
            "path":
                str(
                    EFFECTIVE_SCHEMA_PATH.relative_to(
                        REPO
                    )
                ),

            "sha256":
                effective_schema_sha,

            "size_bytes":
                int(
                    EFFECTIVE_SCHEMA_PATH.stat().st_size
                ),
        },
        {
            "path":
                str(
                    RECEIPT_PATH.relative_to(
                        REPO
                    )
                ),

            "sha256":
                receipt_sha,

            "size_bytes":
                int(
                    RECEIPT_PATH.stat().st_size
                ),
        },
    ],
}


atomic_json(
    MANIFEST_PATH,
    manifest_payload,
)


manifest_sha = sha256_file(
    MANIFEST_PATH
)


print(
    "Erratum:"
)

print(
    " ",
    ERRATUM_PATH.relative_to(
        REPO
    )
)

print(
    " SHA256:",
    erratum_sha
)


print(
    "\nEffective schema:"
)

print(
    " ",
    EFFECTIVE_SCHEMA_PATH.relative_to(
        REPO
    )
)

print(
    " SHA256:",
    effective_schema_sha
)


print(
    "\nReceipt:"
)

print(
    " ",
    RECEIPT_PATH.relative_to(
        REPO
    )
)

print(
    " SHA256:",
    receipt_sha
)


print(
    "\nManifest:"
)

print(
    " ",
    MANIFEST_PATH.relative_to(
        REPO
    )
)

print(
    " SHA256:",
    manifest_sha
)


# =============================================================================
# 14. PRE-COMMIT GIT AUDIT
# =============================================================================

banner(
    "STAGE26-8D2A :: PRE-COMMIT GIT AUDIT"
)


status_lines = [
    line
    for line in git(
        "status",
        "--porcelain",
    ).splitlines()
    if line.strip()
]


for line in status_lines:

    print(
        line
    )


expected_prefix = (
    "?? "
    +
    str(
        OUT_DIR.relative_to(
            REPO
        )
    )
)


if not status_lines:

    raise RuntimeError(
        "No Stage26-8D2A files detected by Git."
    )


if any(
    not line.startswith(
        expected_prefix
    )
    for line in status_lines
):

    raise RuntimeError(
        "Unexpected repository modification detected "
        "before Stage26-8D2A commit."
    )


# =============================================================================
# 15. COMMIT
# =============================================================================

banner(
    "STAGE26-8D2A :: COMMIT"
)


git(
    "add",
    str(
        OUT_DIR.relative_to(
            REPO
        )
    ),
)


staged = git(
    "diff",
    "--cached",
    "--name-only",
).splitlines()


expected_staged = sorted(
    [
        str(
            ERRATUM_PATH.relative_to(
                REPO
            )
        ),
        str(
            EFFECTIVE_SCHEMA_PATH.relative_to(
                REPO
            )
        ),
        str(
            RECEIPT_PATH.relative_to(
                REPO
            )
        ),
        str(
            MANIFEST_PATH.relative_to(
                REPO
            )
        ),
    ]
)


print(
    "Staged files:"
)


for path in staged:

    print(
        " ",
        path
    )


if sorted(
    staged
) != expected_staged:

    raise RuntimeError(
        "Stage26-8D2A staged-file set differs "
        "from the expected four files."
    )


git(
    "commit",
    "-m",
    COMMIT_MESSAGE,
)


new_head = git(
    "rev-parse",
    "HEAD",
)

parent = git(
    "rev-parse",
    "HEAD^",
)

subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "Parent :",
    parent
)

print(
    "HEAD   :",
    new_head
)

print(
    "Subject:",
    subject
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage26-8D2A commit parent mismatch."
    )


if subject != COMMIT_MESSAGE:

    raise RuntimeError(
        "Unexpected Stage26-8D2A commit subject."
    )


# =============================================================================
# 16. AUTHENTICATED PUSH
# =============================================================================

banner(
    "STAGE26-8D2A :: PUSH"
)


try:

    from kaggle_secrets import UserSecretsClient

except Exception as exc:

    raise RuntimeError(
        "Kaggle UserSecretsClient unavailable."
    ) from exc


try:

    github_token = (
        UserSecretsClient()
        .get_secret(
            "GITHUB_TOKEN"
        )
    )

except Exception as exc:

    raise RuntimeError(
        "Could not read Kaggle Secret GITHUB_TOKEN. "
        "Do not paste the token into notebook source."
    ) from exc


if (
    not github_token
    or
    len(
        github_token.strip()
    )
    <
    20
):

    raise RuntimeError(
        "GITHUB_TOKEN secret is empty or unusable."
    )


askpass_fd, askpass_name = tempfile.mkstemp(
    prefix="stage26_8d2a_askpass_",
    suffix=".sh",
)


os.close(
    askpass_fd
)


askpass_path = Path(
    askpass_name
)


try:

    askpass_path.write_text(
        "#!/bin/sh\n"
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *Password*) printf "%s\\n" "$STAGE26_GITHUB_TOKEN" ;;\n'
        '  *) printf "%s\\n" "" ;;\n'
        "esac\n",
        encoding="utf-8",
    )


    askpass_path.chmod(
        askpass_path.stat().st_mode
        |
        stat.S_IXUSR
        |
        stat.S_IXGRP
        |
        stat.S_IXOTH
    )


    push_env = os.environ.copy()


    push_env[
        "GIT_ASKPASS"
    ] = str(
        askpass_path
    )


    push_env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"


    push_env[
        "STAGE26_GITHUB_TOKEN"
    ] = github_token.strip()


    push_output = git(
        "push",
        "origin",
        "main",
        env=push_env,
    )


    print(
        push_output
        if push_output
        else
        "<push completed>"
    )


finally:

    try:

        askpass_path.unlink(
            missing_ok=True
        )

    except Exception:

        pass


    github_token = None


    if "push_env" in locals():

        push_env.pop(
            "STAGE26_GITHUB_TOKEN",
            None,
        )


# =============================================================================
# 17. REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-8D2A :: REMOTE BYTE VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_after = git(
    "rev-parse",
    "HEAD",
)

remote_after = git(
    "rev-parse",
    "origin/main",
)


print(
    "Local HEAD :",
    local_after
)

print(
    "origin/main:",
    remote_after
)


if (
    local_after != new_head
    or
    remote_after != new_head
):

    raise RuntimeError(
        "Local/remote commit mismatch "
        "after Stage26-8D2A push."
    )


verification = []


for path in [
    ERRATUM_PATH,
    EFFECTIVE_SCHEMA_PATH,
    RECEIPT_PATH,
    MANIFEST_PATH,
]:

    rel = str(
        path.relative_to(
            REPO
        )
    )


    local_bytes = path.read_bytes()

    remote_bytes = git_blob_bytes(
        "origin/main",
        rel,
    )


    local_sha = sha256_bytes(
        local_bytes
    )

    remote_sha = sha256_bytes(
        remote_bytes
    )


    same = (
        local_bytes
        ==
        remote_bytes
    )


    verification.append(
        {
            "path":
                rel,

            "local_sha256":
                local_sha,

            "remote_sha256":
                remote_sha,

            "byte_identical":
                same,
        }
    )


    print(
        f"{'PASS' if same else 'FAIL'} "
        f"{rel}"
    )

    print(
        "  local :",
        local_sha
    )

    print(
        "  remote:",
        remote_sha
    )


    if not same:

        raise RuntimeError(
            "Remote byte verification failed:\n"
            f"{rel}"
        )


# =============================================================================
# 18. FINAL CLEAN-REPO CLOSURE
# =============================================================================

final_status = git(
    "status",
    "--porcelain",
)


banner(
    "STAGE26-8D2A COMPLETE"
)


print(
    "Durable erratum anchor       :",
    new_head
)

print(
    "HEAD == origin/main          :",
    (
        local_after
        ==
        remote_after
        ==
        new_head
    )
)

print(
    "Repo clean                   :",
    final_status == ""
)

print(
    "Remote files byte-identical  :",
    all(
        item[
            "byte_identical"
        ]
        for item in verification
    )
)


print(
    "\nSCHEMA RESOLUTION:"
)

print(
    "  Original Stage26-8D2       : RETAINED UNMODIFIED"
)

print(
    "  Stage26-8D2A               : SEMANTIC ERRATUM"
)

print(
    "  Effective downstream schema: 8D2 + 8D2A"
)


print(
    "\nCORRECTIONS:"
)

print(
    "  T2 memory geometry         : target x CPU x batch x metric"
)

print(
    "  T4 batch scaling           : separate CPU1 / CPU2 multipliers"
)

print(
    "  T5 component table         : boundary / availability matrix"
)

print(
    "  T6 ratio direction         : inference / representation"
)

print(
    "  T6 ratio values            : UNCHANGED / NOT INVERTED"
)


print(
    "\nSCIENTIFIC STATE:"
)

print(
    "  measured values changed    : NO"
)

print(
    "  timing                     : NO"
)

print(
    "  inference                  : NO"
)

print(
    "  model loading              : NO"
)

print(
    "  bootstrap                  : NO"
)

print(
    "  random sampling            : NO"
)

print(
    "  memory aggregation         : NO"
)

print(
    "  ratio recomputation        : NO"
)

print(
    "  ratio inversion            : NO"
)

print(
    "  Stage26-7C                 : UNCHANGED"
)

print(
    "  Pareto                     : UNCHANGED"
)

print(
    "  PCAP                       : NO"
)

print(
    "  labels                     : NO"
)

print(
    "  GPU                        : OFF"
)


if final_status:

    raise RuntimeError(
        "Repository is not clean after Stage26-8D2A."
    )


print(
    "\nNEXT:"
)

print(
    "  Generate the eight CPU publication tables using"
)

print(
    "  stage26_8d2a_effective_cpu_publication_schema.json."
)

print(
    "  GPU remains OFF."
)


STAGE26-8D2A :: DURABLE STATE GATE
Expected parent: f04591864f2c64096a3932bc6f793e84a36e81f2
Local HEAD     : f04591864f2c64096a3932bc6f793e84a36e81f2
origin/main    : f04591864f2c64096a3932bc6f793e84a36e81f2
Repo clean     : True

STAGE26-8D2A :: VERIFY ORIGINAL 8D2 FREEZE
stage26_8d2_cpu_publication_schema.json
  expected: f5d66a399ef66f9997de496f109e5ec48c55262b4cd6d7a88e570e8ab7e3cdd6
  actual  : f5d66a399ef66f9997de496f109e5ec48c55262b4cd6d7a88e570e8ab7e3cdd6
stage26_8d2_cpu_publication_schema_receipt.json
  expected: 63a7b92fef7393457c2540d07cd0b654ef0b22aa718ac44eebb4e8e7d84980e2
  actual  : 63a7b92fef7393457c2540d07cd0b654ef0b22aa718ac44eebb4e8e7d84980e2
stage26_8d2_cpu_publication_schema_manifest.json
  expected: ddc35d05c8e9470f1b90d9eb7f620aafbb359e300418e5bf50e85ce171ac441c
  actual  : ddc35d05c8e9470f1b90d9eb7f620aafbb359e300418e5bf50e85ce171ac441c

Original schema table count: 8
Original schema figure count: 8

STAGE26-8D2A :: SOURCE BYTE IDENTITY
PASS results/stage26_de

456### =============================================================================
# STAGE26-8D3
# GENERATE EIGHT CPU PUBLICATION TABLES FROM THE EFFECTIVE 8D2+8D2A SCHEMA
#
# SCIENTIFIC PARENT:
#   80f7fce76b3a859ea699e8838eced34e9936d421
#
# PURPOSE
# -------
# Materialize the eight frozen CPU publication tables using only already-
# committed Stage26 artifacts and the effective Stage26-8D2A schema.
#
# This cell performs deterministic publication-layer transforms only:
#   - source-column selection / renaming
#   - exact-key joins
#   - bytes -> MiB unit conversion for frozen Stage26-3B memory metrics
#   - representation-only status labels from frozen repetition counts
#   - fingerprint-equality counting required by the frozen sensitivity table
#
# It DOES NOT:
#   - run timing
#   - run inference or load models
#   - bootstrap
#   - resample or use randomness
#   - recompute Stage26-7C ratios/scaling
#   - invert Group-B ratios
#   - recompute Pareto membership
#   - aggregate memory across CPU modes or batch sizes
#   - use cold-start confidence intervals
#   - use historical Stage26-2 confidence intervals
#   - access PCAP or labels
#   - use GPU
#
# OUTPUT
# ------
# results/stage26_deployment_profiling/stage26_8d3_cpu_publication_tables/
#   T26_CPU_WARM_INFERENCE.csv
#   T26_CPU_MEMORY_PACKAGE.csv
#   T26_CPU_COLD_START.csv
#   T26_CPU_CAPACITY_SCALING.csv
#   T26_COMPONENT_MEASUREMENTS.csv
#   T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO.csv
#   T26_PARETO.csv
#   T26_REPRESENTATION_SENSITIVITY.csv
#   stage26_8d3_cpu_publication_tables_index.json
#   stage26_8d3_cpu_publication_tables_receipt.json
#   stage26_8d3_cpu_publication_tables_manifest.json
# =============================================================================

from __future__ import annotations

import csv
import hashlib
import json
import os
import stat
import subprocess
import tempfile
from collections import defaultdict
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from pathlib import Path


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")
RESULT_ROOT = REPO / "results" / "stage26_deployment_profiling"

EXPECTED_PARENT = "80f7fce76b3a859ea699e8838eced34e9936d421"

EFFECTIVE_SCHEMA = (
    RESULT_ROOT
    / "stage26_8d2a_cpu_publication_schema_erratum"
    / "stage26_8d2a_effective_cpu_publication_schema.json"
)
EXPECTED_EFFECTIVE_SCHEMA_SHA256 = (
    "973cc58ed6572d30b1cce94f3659226d99b185390c2aca92d9ed030e50358ad0"
)

PUBLICATION_LABEL = (
    "COMPONENT_LEVEL_WITH_STAGE26_8_4C3_SENSITIVITY_AUDIT_COMPLETED"
)

TARGET_ORDER = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
    "ENS_LGBM_XGB_EQUAL",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
]

TARGET_RANK = {target: i for i, target in enumerate(TARGET_ORDER)}
CPU_ORDER = {
    "CPU_1_PHYSICAL_CORE": 0,
    "CPU_2_PHYSICAL_CORE": 1,
}
BATCH_ORDER = [1, 64, 256, 1024, 8192]
BATCH_RANK = {batch: i for i, batch in enumerate(BATCH_ORDER)}

COLD_COMPONENT_ORDER = [
    "process_spawn_to_worker_ready_ns",
    "framework_import_ns",
    "input_preparation_ns",
    "model_deserialization_load_ns",
    "first_prediction_ns",
    "spawn_to_first_output_ns",
    "parent_process_total_ns",
]
COLD_COMPONENT_RANK = {
    component: i for i, component in enumerate(COLD_COMPONENT_ORDER)
}

# -----------------------------------------------------------------------------
# Source files
# -----------------------------------------------------------------------------

WARM_STATUS = (
    RESULT_ROOT
    / "stage26_2_cpu_warm_inference"
    / "stage26_2_condition_status.csv"
)

WARM_CORRECTED_UNCERTAINTY = (
    RESULT_ROOT
    / "stage26_6f1_bootstrap_corrected_uncertainty"
    / "stage26_6f1_corrected_timing_uncertainty.csv"
)

COLD_IMPLEMENTATION = (
    RESULT_ROOT
    / "stage26_1_cpu_cold_start"
    / "stage26_1_cold_start_implementation.json"
)

COLD_SUMMARY = (
    RESULT_ROOT
    / "stage26_1_cpu_cold_start"
    / "stage26_1_cold_start_summary.csv"
)

MEMORY_SUMMARY = (
    RESULT_ROOT
    / "stage26_3b_cpu_memory_profile"
    / "stage26_3_memory_summary.csv"
)

PACKAGE_SIZES = (
    RESULT_ROOT
    / "stage26_3b_cpu_memory_profile"
    / "stage26_3_package_sizes.csv"
)

CAPACITY_BATCH = (
    RESULT_ROOT
    / "stage26_7c_cpu_capacity_scaling"
    / "stage26_7c_inference_batch_scaling.csv"
)

CAPACITY_TWO_CORE = (
    RESULT_ROOT
    / "stage26_7c_cpu_capacity_scaling"
    / "stage26_7c_two_core_scaling.csv"
)

GROUP_B_CAPACITY = (
    RESULT_ROOT
    / "stage26_7c_cpu_capacity_scaling"
    / "stage26_7c_group_b_component_capacity.csv"
)

PARETO = (
    RESULT_ROOT
    / "stage26_6d_cpu_pareto_point_estimate"
    / "stage26_6d_point_estimate_frontiers.csv"
)

REP_SENS_BATCH = (
    RESULT_ROOT
    / "stage26_8c1_paired_representation_sensitivity"
    / "evidence"
    / "stage26_8c1_batch_sensitivity_summary.csv"
)

REP_SENS_PAIRS = (
    RESULT_ROOT
    / "stage26_8c1_paired_representation_sensitivity"
    / "evidence"
    / "stage26_8c1_pair_results.csv"
)

REP_SENS_AUDIT = (
    RESULT_ROOT
    / "stage26_8c1_paired_representation_sensitivity"
    / "stage26_8c1_sensitivity_audit.json"
)

E2E_DECISION = (
    RESULT_ROOT
    / "stage26_5a_e2e_availability_closure"
    / "stage26_5a_e2e_availability_decision.json"
)

RAW_EXTRACTION_SUMMARY = (
    RESULT_ROOT
    / "stage26_4b3_cpu1_extraction_timing"
    / "stage26_4b3_cpu1_extraction_summary.json"
)

REPRESENTATION_HISTORICAL_SUMMARY = (
    RESULT_ROOT
    / "stage26_4c3_cpu1_representation_timing"
    / "stage26_4c3_cpu1_representation_summary.csv"
)

# -----------------------------------------------------------------------------
# Historical source anchors
# -----------------------------------------------------------------------------

SOURCE_SPECS = [
    (
        "warm_cpu_point_estimates",
        WARM_STATUS,
        "7ffdba3f4ca4ea5cc53097d62aaf27957009b9f6",
    ),
    (
        "corrected_warm_cpu_uncertainty",
        WARM_CORRECTED_UNCERTAINTY,
        "a484148cd8ea7d7c89604d3a015f73bd6f1bc81a",
    ),
    (
        "cold_start_implementation",
        COLD_IMPLEMENTATION,
        "46379b6d036008db4d60b056a66f4c01383e3298",
    ),
    (
        "cold_start_point_estimates",
        COLD_SUMMARY,
        "46379b6d036008db4d60b056a66f4c01383e3298",
    ),
    (
        "cpu_memory_summary",
        MEMORY_SUMMARY,
        "347d93f21d454cc5bda2c45889c890c67cdf0ecc",
    ),
    (
        "deployment_package_sizes",
        PACKAGE_SIZES,
        "347d93f21d454cc5bda2c45889c890c67cdf0ecc",
    ),
    (
        "capacity_batch_scaling",
        CAPACITY_BATCH,
        "9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3",
    ),
    (
        "capacity_two_core_scaling",
        CAPACITY_TWO_CORE,
        "9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3",
    ),
    (
        "group_b_component_capacity",
        GROUP_B_CAPACITY,
        "9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3",
    ),
    (
        "pareto_point_estimate",
        PARETO,
        "ff9d329785c6cd30d273729356f402e59dc4e844",
    ),
    (
        "representation_sensitivity_batch_summary",
        REP_SENS_BATCH,
        "fd4497e333ebeb72b1eb188ed64ca2cdd0f08567",
    ),
    (
        "representation_sensitivity_pair_results",
        REP_SENS_PAIRS,
        "fd4497e333ebeb72b1eb188ed64ca2cdd0f08567",
    ),
    (
        "representation_sensitivity_audit",
        REP_SENS_AUDIT,
        "fd4497e333ebeb72b1eb188ed64ca2cdd0f08567",
    ),
    (
        "complete_e2e_closure",
        E2E_DECISION,
        "92495eac2c9202973ff4d889e3c12669c06e9ef1",
    ),
    (
        "raw_extraction_component",
        RAW_EXTRACTION_SUMMARY,
        "55aba2e1cc08385659479c6275234dc23b11d231",
    ),
    (
        "historical_representation_component",
        REPRESENTATION_HISTORICAL_SUMMARY,
        "aee59fabab2465cca7c80904279bb6d3ef23894f",
    ),
]

# -----------------------------------------------------------------------------
# Output
# -----------------------------------------------------------------------------

OUT_DIR = RESULT_ROOT / "stage26_8d3_cpu_publication_tables"

TABLE_PATHS = {
    "T26_CPU_WARM_INFERENCE": OUT_DIR / "T26_CPU_WARM_INFERENCE.csv",
    "T26_CPU_MEMORY_PACKAGE": OUT_DIR / "T26_CPU_MEMORY_PACKAGE.csv",
    "T26_CPU_COLD_START": OUT_DIR / "T26_CPU_COLD_START.csv",
    "T26_CPU_CAPACITY_SCALING": OUT_DIR / "T26_CPU_CAPACITY_SCALING.csv",
    "T26_COMPONENT_MEASUREMENTS": OUT_DIR / "T26_COMPONENT_MEASUREMENTS.csv",
    "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO": (
        OUT_DIR / "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO.csv"
    ),
    "T26_PARETO": OUT_DIR / "T26_PARETO.csv",
    "T26_REPRESENTATION_SENSITIVITY": (
        OUT_DIR / "T26_REPRESENTATION_SENSITIVITY.csv"
    ),
}

INDEX_PATH = OUT_DIR / "stage26_8d3_cpu_publication_tables_index.json"
RECEIPT_PATH = OUT_DIR / "stage26_8d3_cpu_publication_tables_receipt.json"
MANIFEST_PATH = OUT_DIR / "stage26_8d3_cpu_publication_tables_manifest.json"

COMMIT_MESSAGE = "stage26: generate CPU publication tables"


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):
    print("\n" + "=" * 124)
    print(text)
    print("=" * 124)


def run(cmd, *, cwd=REPO, check=True, env=None):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
        env=env,
    )
    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}\n{p.stdout}"
        )
    return p.stdout.strip()


def git(*args, check=True, env=None):
    return run(["git", *args], check=check, env=env)


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            block = f.read(8 * 1024 * 1024)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def read_csv(path):
    with Path(path).open("r", encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, sort_keys=True, allow_nan=False)
        f.write("\n")
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def atomic_csv(path, columns, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with tmp.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=columns,
            extrasaction="raise",
            lineterminator="\n",
        )
        writer.writeheader()
        for row in rows:
            missing = [column for column in columns if column not in row]
            if missing:
                raise RuntimeError(
                    f"Row for {path.name} is missing columns: {missing}"
                )
            writer.writerow({column: row[column] for column in columns})
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def git_blob_bytes(ref, rel_path):
    p = subprocess.run(
        ["git", "show", f"{ref}:{rel_path}"],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )
    if p.returncode != 0:
        raise RuntimeError(
            f"Could not read {rel_path} from {ref}:\n"
            + p.stderr.decode("utf-8", errors="replace")
        )
    return p.stdout


def verify_historical_identity(label, path, anchor):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    rel = str(path.relative_to(REPO))
    current = path.read_bytes()
    historical = git_blob_bytes(anchor, rel)
    current_sha = sha256_bytes(current)
    historical_sha = sha256_bytes(historical)
    same = current == historical
    print(f"{'PASS' if same else 'FAIL'} {label}")
    print("  path      :", rel)
    print("  anchor    :", anchor)
    print("  historical:", historical_sha)
    print("  current   :", current_sha)
    if not same:
        raise RuntimeError(
            f"Source changed since durable scientific anchor: {rel}"
        )
    return {
        "label": label,
        "path": rel,
        "anchor": anchor,
        "sha256": current_sha,
        "byte_identical_to_anchor": True,
    }


def table_spec(schema, table_id):
    matches = [item for item in schema["tables"] if item["id"] == table_id]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one effective table spec for {table_id}; found {len(matches)}."
        )
    return matches[0]


def index_unique(rows, key, label):
    out = {}
    for row in rows:
        value = row[key]
        if value in out:
            raise RuntimeError(f"Duplicate {label} key: {value}")
        out[value] = row
    return out


def parse_int(value, label):
    try:
        return int(value)
    except Exception as exc:
        raise RuntimeError(f"Invalid integer for {label}: {value!r}") from exc


def parse_bool(value, label):
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text == "true":
        return True
    if text == "false":
        return False
    raise RuntimeError(f"Invalid boolean for {label}: {value!r}")


def decimal_value(value, label):
    text = str(value).strip()
    if text == "":
        return None
    try:
        return Decimal(text)
    except InvalidOperation as exc:
        raise RuntimeError(f"Invalid decimal for {label}: {value!r}") from exc


def numeric_equal(a, b, label):
    da = decimal_value(a, label + " left")
    db = decimal_value(b, label + " right")
    if da is None or db is None:
        return da is None and db is None
    return da == db


def bytes_to_mib_text(value, label):
    text = str(value).strip()
    if text == "":
        return ""
    d = decimal_value(text, label)
    out = d / Decimal(1048576)
    # Deterministic decimal rendering; conversion only, no aggregation.
    rendered = format(out, ".12f").rstrip("0").rstrip(".")
    return rendered if rendered else "0"


def target_sort_key(target_id):
    if target_id not in TARGET_RANK:
        raise RuntimeError(f"Unexpected target_id: {target_id}")
    return TARGET_RANK[target_id]


def batch_sort_key(batch_size):
    batch = int(batch_size)
    if batch not in BATCH_RANK:
        raise RuntimeError(f"Unexpected batch size: {batch}")
    return BATCH_RANK[batch]


def memory_status_from_counts(row):
    planned = parse_int(row["planned_repetitions"], "planned_repetitions")
    passed = parse_int(row["pass_repetitions"], "pass_repetitions")
    oom = parse_int(
        row["resource_limit_oom_repetitions"],
        "resource_limit_oom_repetitions",
    )
    timeout = parse_int(
        row["timeout_resource_limit_repetitions"],
        "timeout_resource_limit_repetitions",
    )

    if passed + oom + timeout != planned:
        raise RuntimeError(
            "Stage26-3B repetition counts do not exhaust the frozen condition: "
            f"{row.get('condition_id')}"
        )

    if passed == planned and oom == 0 and timeout == 0:
        return "PASS", "NONE"
    if oom == planned and passed == 0 and timeout == 0:
        return "RESOURCE_LIMIT_OOM", "RESOURCE_LIMIT_OOM"
    if timeout == planned and passed == 0 and oom == 0:
        return "TIMEOUT_RESOURCE_LIMIT", "TIMEOUT_RESOURCE_LIMIT"

    outcomes = []
    if passed:
        outcomes.append(f"PASS:{passed}")
    if oom:
        outcomes.append(f"RESOURCE_LIMIT_OOM:{oom}")
    if timeout:
        outcomes.append(f"TIMEOUT_RESOURCE_LIMIT:{timeout}")
    return "MIXED_FROZEN_OUTCOME", ";".join(outcomes)


# =============================================================================
# 2. DURABLE STATE GATE
# =============================================================================

banner("STAGE26-8D3 :: DURABLE STATE GATE")

git("fetch", "origin", "main")

head = git("rev-parse", "HEAD")
remote = git("rev-parse", "origin/main")
status = git("status", "--porcelain")

print("Expected parent:", EXPECTED_PARENT)
print("Local HEAD     :", head)
print("origin/main    :", remote)
print("Repo clean     :", status == "")

if head != EXPECTED_PARENT:
    raise RuntimeError("Unexpected local HEAD before Stage26-8D3.")
if remote != EXPECTED_PARENT:
    raise RuntimeError("origin/main changed before Stage26-8D3.")
if status:
    raise RuntimeError("Repository must be clean before Stage26-8D3.")
if OUT_DIR.exists():
    raise RuntimeError(
        f"Stage26-8D3 output directory already exists: {OUT_DIR}\n"
        "Do not overwrite durable publication tables."
    )


# =============================================================================
# 3. EFFECTIVE SCHEMA GATE
# =============================================================================

banner("STAGE26-8D3 :: EFFECTIVE SCHEMA GATE")

if not EFFECTIVE_SCHEMA.is_file():
    raise FileNotFoundError(EFFECTIVE_SCHEMA)

schema_sha = sha256_file(EFFECTIVE_SCHEMA)
print("Expected effective schema SHA256:", EXPECTED_EFFECTIVE_SCHEMA_SHA256)
print("Actual effective schema SHA256  :", schema_sha)

if schema_sha != EXPECTED_EFFECTIVE_SCHEMA_SHA256:
    raise RuntimeError("Effective Stage26-8D2A schema SHA mismatch.")

schema = read_json(EFFECTIVE_SCHEMA)

expected_table_ids = list(TABLE_PATHS)
actual_table_ids = [item["id"] for item in schema["tables"]]

print("Table IDs:")
for table_id in actual_table_ids:
    print(" ", table_id)

if actual_table_ids != expected_table_ids:
    raise RuntimeError(
        "Effective schema table identity/order differs from the frozen eight-table universe."
    )

if schema.get("publication_label") != PUBLICATION_LABEL:
    raise RuntimeError("Unexpected effective publication label.")

if schema["scope"].get("gpu_allowed") is not False:
    raise RuntimeError("GPU must remain OFF for Stage26-8D3.")


# =============================================================================
# 4. SOURCE BYTE-IDENTITY GATES
# =============================================================================

banner("STAGE26-8D3 :: SOURCE BYTE-IDENTITY GATES")

source_identity = []
for label, path, anchor in SOURCE_SPECS:
    source_identity.append(
        verify_historical_identity(label, path, anchor)
    )


# =============================================================================
# 5. LOAD SMALL COMMITTED SUMMARY ARTIFACTS ONLY
# =============================================================================

banner("STAGE26-8D3 :: LOAD COMMITTED SUMMARY ARTIFACTS")

warm_status_rows = read_csv(WARM_STATUS)
warm_uncertainty_rows = read_csv(WARM_CORRECTED_UNCERTAINTY)
cold_impl = read_json(COLD_IMPLEMENTATION)
cold_rows = read_csv(COLD_SUMMARY)
memory_rows = read_csv(MEMORY_SUMMARY)
package_rows = read_csv(PACKAGE_SIZES)
capacity_batch_rows = read_csv(CAPACITY_BATCH)
capacity_two_core_rows = read_csv(CAPACITY_TWO_CORE)
group_b_rows = read_csv(GROUP_B_CAPACITY)
pareto_rows = read_csv(PARETO)
rep_batch_rows = read_csv(REP_SENS_BATCH)
rep_pair_rows = read_csv(REP_SENS_PAIRS)
rep_audit = read_json(REP_SENS_AUDIT)

print("Warm status rows              :", len(warm_status_rows))
print("Warm uncertainty rows         :", len(warm_uncertainty_rows))
print("Cold-start summary rows       :", len(cold_rows))
print("Memory condition rows         :", len(memory_rows))
print("Package-size rows             :", len(package_rows))
print("Capacity batch rows           :", len(capacity_batch_rows))
print("Capacity two-core rows        :", len(capacity_two_core_rows))
print("Group-B component-ratio rows  :", len(group_b_rows))
print("Pareto rows                   :", len(pareto_rows))
print("Sensitivity batch rows        :", len(rep_batch_rows))
print("Sensitivity pair rows         :", len(rep_pair_rows))

if len(warm_status_rows) != 80:
    raise RuntimeError("Expected exactly 80 Stage26-2 CPU conditions.")
if len(memory_rows) != 80:
    raise RuntimeError("Expected exactly 80 Stage26-3B memory conditions.")
if len(package_rows) != 8:
    raise RuntimeError("Expected exactly 8 Stage26-3B package-size rows.")
if len(capacity_batch_rows) != 80:
    raise RuntimeError("Expected exactly 80 Stage26-7C batch-scaling rows.")
if len(rep_batch_rows) != 5:
    raise RuntimeError("Expected exactly five frozen Stage26-8 sensitivity batches.")
if len(rep_pair_rows) != 25:
    raise RuntimeError("Expected exactly 25 frozen Stage26-8 paired blocks.")


# =============================================================================
# 6. COMMON TARGET/GROUP MAP
# =============================================================================

banner("STAGE26-8D3 :: COMMON TARGET/GROUP MAP")

groups_seen = defaultdict(set)
for row in warm_status_rows:
    groups_seen[row["target_id"]].add(row["comparison_group"])

if set(groups_seen) != set(TARGET_ORDER):
    raise RuntimeError(
        "Warm-inference target universe differs from the frozen eight-target universe."
    )

group_by_target = {}
for target in TARGET_ORDER:
    values = groups_seen[target]
    if len(values) != 1:
        raise RuntimeError(f"Target has non-unique comparison group: {target} -> {values}")
    group_by_target[target] = next(iter(values))
    print(f"{target:44s} -> {group_by_target[target]}")


# =============================================================================
# 7. T1 — CPU ISOLATED WARM INFERENCE
# =============================================================================

banner("STAGE26-8D3 :: BUILD T26_CPU_WARM_INFERENCE")

warm_unc_by_condition = index_unique(
    warm_uncertainty_rows,
    "condition_id",
    "warm corrected uncertainty",
)

T1 = []

for src in warm_status_rows:
    condition_id = src["condition_id"]
    status_value = src["status"]
    unc = warm_unc_by_condition.get(condition_id)

    if status_value == "PASS" and unc is None:
        raise RuntimeError(
            f"PASS warm condition lacks Stage26-6F1 corrected uncertainty: {condition_id}"
        )

    if unc is not None:
        identity_fields = [
            ("target_id", "target_id"),
            ("comparison_group", "comparison_group"),
            ("hardware_mode", "hardware_mode"),
            ("batch_size", "batch_size"),
            ("status", "status"),
        ]
        for left, right in identity_fields:
            if src[left] != unc[right]:
                raise RuntimeError(
                    f"Warm Stage26-2/6F1 identity mismatch for {condition_id}: "
                    f"{left}={src[left]!r} vs {unc[right]!r}"
                )

        if src["timed_runs_observed"] != unc["n"]:
            raise RuntimeError(
                f"Warm observation-count mismatch for {condition_id}."
            )

        point_pairs = [
            ("p50_batch_latency_ms", "p50_point_ms"),
            ("p95_batch_latency_ms", "p95_point_ms"),
            ("p99_batch_latency_ms", "p99_point_ms_if_n_gte_100"),
            ("median_throughput_flows_per_second", "median_throughput_point"),
        ]
        for a, b in point_pairs:
            if not numeric_equal(src[a], unc[b], f"{condition_id}:{a}"):
                raise RuntimeError(
                    f"Warm point estimate changed between Stage26-2 and 6F1: "
                    f"{condition_id} {a}"
                )

        if status_value == "PASS" and unc["corrected_ci_status"] != (
            "PROTOCOL_CERTIFIED_DERIVED_UNCERTAINTY"
        ):
            raise RuntimeError(
                f"Unexpected Stage26-6F1 corrected CI status for {condition_id}: "
                f"{unc['corrected_ci_status']}"
            )

    def u(field):
        return "" if unc is None else unc.get(field, "")

    T1.append(
        {
            "target_id": src["target_id"],
            "group": src["comparison_group"],
            "cpu_mode": src["hardware_mode"],
            "batch_size": src["batch_size"],
            "status": status_value,
            "n_timed_observations": src["timed_runs_observed"],
            "p50_latency_ms": src["p50_batch_latency_ms"],
            "p50_ci95_low_ms": u("corrected_p50_ci_low_ms"),
            "p50_ci95_high_ms": u("corrected_p50_ci_high_ms"),
            "p95_latency_ms": src["p95_batch_latency_ms"],
            "p95_ci95_low_ms": u("corrected_p95_ci_low_ms"),
            "p95_ci95_high_ms": u("corrected_p95_ci_high_ms"),
            "p99_latency_ms": src["p99_batch_latency_ms"],
            "p99_ci95_low_ms": u("corrected_p99_ci_low_ms_if_n_gte_100"),
            "p99_ci95_high_ms": u("corrected_p99_ci_high_ms_if_n_gte_100"),
            "median_throughput_samples_per_s": (
                src["median_throughput_flows_per_second"]
            ),
            "throughput_ci95_low_samples_per_s": (
                u("corrected_median_throughput_ci_low")
            ),
            "throughput_ci95_high_samples_per_s": (
                u("corrected_median_throughput_ci_high")
            ),
            "resource_limit_outcome": (
                "NONE" if status_value == "PASS" else status_value
            ),
        }
    )

T1.sort(
    key=lambda row: (
        target_sort_key(row["target_id"]),
        CPU_ORDER[row["cpu_mode"]],
        batch_sort_key(row["batch_size"]),
    )
)

if len(T1) != 80:
    raise RuntimeError("T26_CPU_WARM_INFERENCE must contain 80 rows.")


# =============================================================================
# 8. T2 — CPU MEMORY + DEPLOYMENT PACKAGE
# =============================================================================

banner("STAGE26-8D3 :: BUILD T26_CPU_MEMORY_PACKAGE")

package_by_target = index_unique(package_rows, "target_id", "package size")

if set(package_by_target) != set(TARGET_ORDER):
    raise RuntimeError("Package-size target universe mismatch.")

memory_spec = table_spec(schema, "T26_CPU_MEMORY_PACKAGE")
metric_specs = memory_spec["frozen_primary_memory_metrics"]
metric_rank = {
    item["output_name"]: i for i, item in enumerate(metric_specs)
}

T2 = []

for src in memory_rows:
    target = src["target_id"]
    if target not in package_by_target:
        raise RuntimeError(f"Memory target lacks package-size row: {target}")
    if src["comparison_group"] != group_by_target[target]:
        raise RuntimeError(f"Memory comparison-group mismatch for {target}.")

    memory_status, resource_outcome = memory_status_from_counts(src)

    for metric in metric_specs:
        source_field = metric["source_field"]
        output_name = metric["output_name"]
        if source_field not in src:
            raise RuntimeError(
                f"Frozen memory source field missing: {source_field}"
            )

        T2.append(
            {
                "target_id": target,
                "group": src["comparison_group"],
                "cpu_mode": src["hardware_mode"],
                "batch_size": src["batch_size"],
                "deployment_package_size_mib": (
                    package_by_target[target]["deployment_package_size_mib"]
                ),
                "memory_metric_name": output_name,
                "memory_value_mib": bytes_to_mib_text(
                    src[source_field],
                    f"{src['condition_id']}:{source_field}",
                ),
                "planned_repetitions": src["planned_repetitions"],
                "pass_repetitions": src["pass_repetitions"],
                "resource_limit_oom_repetitions": (
                    src["resource_limit_oom_repetitions"]
                ),
                "timeout_resource_limit_repetitions": (
                    src["timeout_resource_limit_repetitions"]
                ),
                "memory_status": memory_status,
                "resource_limit_outcome": resource_outcome,
            }
        )

T2.sort(
    key=lambda row: (
        target_sort_key(row["target_id"]),
        CPU_ORDER[row["cpu_mode"]],
        batch_sort_key(row["batch_size"]),
        metric_rank[row["memory_metric_name"]],
    )
)

expected_t2_rows = 80 * len(metric_specs)
if len(T2) != expected_t2_rows:
    raise RuntimeError(
        f"T26_CPU_MEMORY_PACKAGE expected {expected_t2_rows} rows; got {len(T2)}."
    )


# =============================================================================
# 9. T3 — CPU COLD START, POINT ESTIMATES ONLY
# =============================================================================

banner("STAGE26-8D3 :: BUILD T26_CPU_COLD_START")

if cold_impl.get("gpu") is not False:
    raise RuntimeError("Cold-start implementation unexpectedly indicates GPU use.")

cold_cpu_mode = cold_impl["hardware_mode"]
expected_cold_n = int(cold_impl["repetitions_per_target"])

if cold_cpu_mode != "CPU_1_PHYSICAL_CORE":
    raise RuntimeError("Unexpected frozen cold-start hardware mode.")
if expected_cold_n != 20:
    raise RuntimeError("Unexpected frozen cold-start repetition count.")

cold_targets = {row["target_id"] for row in cold_rows}
if cold_targets != set(TARGET_ORDER):
    raise RuntimeError("Cold-start target universe mismatch.")

T3 = []

for src in cold_rows:
    n = parse_int(src["n"], "cold-start n")
    if n != expected_cold_n:
        raise RuntimeError(
            f"Cold-start summary is incomplete for {src['target_id']} / {src['component']}: "
            f"n={n}, expected={expected_cold_n}"
        )

    component = src["component"]
    if component not in COLD_COMPONENT_RANK:
        raise RuntimeError(f"Unexpected cold-start component: {component}")

    T3.append(
        {
            "target_id": src["target_id"],
            "group": group_by_target[src["target_id"]],
            "cpu_mode": cold_cpu_mode,
            "component": component,
            "n_observations": src["n"],
            "p50_ms": src["p50_ms"],
            "p95_ms": src["p95_ms"],
            "status": "PASS",
        }
    )

T3.sort(
    key=lambda row: (
        target_sort_key(row["target_id"]),
        COLD_COMPONENT_RANK[row["component"]],
    )
)

expected_t3_rows = len(TARGET_ORDER) * len(COLD_COMPONENT_ORDER)
if len(T3) != expected_t3_rows:
    raise RuntimeError(
        f"T26_CPU_COLD_START expected {expected_t3_rows} rows; got {len(T3)}."
    )


# =============================================================================
# 10. T4 — CPU CAPACITY / SCALING, IMMUTABLE STAGE26-7C VALUES
# =============================================================================

banner("STAGE26-8D3 :: BUILD T26_CPU_CAPACITY_SCALING")

batch_keyed = {}
for src in capacity_batch_rows:
    target = src["target_id"]
    batch = parse_int(src["batch_size"], "capacity batch")
    mode = src["hardware_mode"]
    if target not in TARGET_RANK:
        raise RuntimeError(f"Unexpected Stage26-7C target: {target}")
    if mode not in CPU_ORDER:
        raise RuntimeError(f"Unexpected Stage26-7C hardware mode: {mode}")
    key = (target, batch, mode)
    if key in batch_keyed:
        raise RuntimeError(f"Duplicate Stage26-7C batch row: {key}")
    batch_keyed[key] = src

expected_batch_keys = {
    (target, batch, mode)
    for target in TARGET_ORDER
    for batch in BATCH_ORDER
    for mode in CPU_ORDER
}

if set(batch_keyed) != expected_batch_keys:
    missing = sorted(expected_batch_keys - set(batch_keyed))
    extra = sorted(set(batch_keyed) - expected_batch_keys)
    raise RuntimeError(
        f"Stage26-7C batch-scaling key universe mismatch. missing={missing[:5]} extra={extra[:5]}"
    )

two_core_by_key = {}
for src in capacity_two_core_rows:
    key = (src["target_id"], parse_int(src["batch_size"], "two-core batch"))
    if key in two_core_by_key:
        raise RuntimeError(f"Duplicate Stage26-7C two-core row: {key}")
    if parse_bool(src["uncertainty_propagated"], "two-core uncertainty"):
        raise RuntimeError("Stage26-7C two-core row unexpectedly propagates uncertainty.")
    two_core_by_key[key] = src

T4 = []

for target in TARGET_ORDER:
    for batch in BATCH_ORDER:
        cpu1 = batch_keyed[(target, batch, "CPU_1_PHYSICAL_CORE")]
        cpu2 = batch_keyed[(target, batch, "CPU_2_PHYSICAL_CORE")]
        two = two_core_by_key.get((target, batch))

        if parse_bool(cpu1["uncertainty_propagated"], "CPU1 batch uncertainty"):
            raise RuntimeError("Stage26-7C CPU1 batch row unexpectedly propagates uncertainty.")
        if parse_bool(cpu2["uncertainty_propagated"], "CPU2 batch uncertainty"):
            raise RuntimeError("Stage26-7C CPU2 batch row unexpectedly propagates uncertainty.")

        both_pass = cpu1["status"] == "PASS" and cpu2["status"] == "PASS"
        if both_pass and two is None:
            raise RuntimeError(
                f"Matched PASS CPU1/CPU2 condition lacks immutable Stage26-7C two-core row: "
                f"{target}, batch={batch}"
            )

        T4.append(
            {
                "target_id": target,
                "group": group_by_target[target],
                "batch_size": str(batch),
                "cpu1_status": cpu1["status"],
                "cpu2_status": cpu2["status"],
                "cpu1_throughput_multiplier_vs_b1": (
                    cpu1["throughput_multiplier_vs_B1"]
                ),
                "cpu2_throughput_multiplier_vs_b1": (
                    cpu2["throughput_multiplier_vs_B1"]
                ),
                "cpu2_over_cpu1_speedup": (
                    "" if two is None else two["two_core_speedup"]
                ),
                "physical_core_parallel_efficiency": (
                    "" if two is None else two["two_core_parallel_efficiency"]
                ),
            }
        )

if len(T4) != 40:
    raise RuntimeError("T26_CPU_CAPACITY_SCALING must contain 40 rows.")


# =============================================================================
# 11. T5 — COMPONENT BOUNDARY / E2E AVAILABILITY MATRIX
# =============================================================================

banner("STAGE26-8D3 :: BUILD T26_COMPONENT_MEASUREMENTS")

component_spec = table_spec(schema, "T26_COMPONENT_MEASUREMENTS")
fixed_component_rows = component_spec.get("fixed_rows")

if not isinstance(fixed_component_rows, list) or len(fixed_component_rows) != 4:
    raise RuntimeError("Effective T26_COMPONENT_MEASUREMENTS fixed-row lock is missing.")

T5 = []
for src in fixed_component_rows:
    row = dict(src)
    if row.get("complete_e2e_available") is not False:
        raise RuntimeError(
            "Component-boundary table unexpectedly allows complete E2E measurement."
        )
    T5.append(row)


# =============================================================================
# 12. T6 — GROUP-B INFERENCE / REPRESENTATION COMPONENT CAPACITY RATIO
# =============================================================================

banner("STAGE26-8D3 :: BUILD T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO")

T6 = []

for src in group_b_rows:
    if src["hardware_mode"] != "CPU_1_PHYSICAL_CORE":
        raise RuntimeError("Group-B component ratio contains unexpected hardware mode.")
    if parse_bool(src["uncertainty_propagated"], "Group-B ratio uncertainty"):
        raise RuntimeError("Group-B component ratio unexpectedly propagates uncertainty.")
    if src["target_id"] not in {
        "STAGE20_MASKED_CNN_V1",
        "STAGE21_MASKED_VIT_V1",
    }:
        raise RuntimeError(f"Unexpected Group-B ratio target: {src['target_id']}")

    T6.append(
        {
            "target_id": src["target_id"],
            "batch_size": src["batch_size"],
            "inference_over_representation_capacity_ratio": (
                src["component_capacity_ratio_inference_over_representation"]
            ),
            "publication_label": PUBLICATION_LABEL,
        }
    )

T6.sort(
    key=lambda row: (
        target_sort_key(row["target_id"]),
        batch_sort_key(row["batch_size"]),
    )
)

if len(T6) != 7:
    raise RuntimeError("Expected exactly 7 frozen Group-B matched component-ratio rows.")


# =============================================================================
# 13. T7 — IMMUTABLE WITHIN-GROUP PARETO POINT ESTIMATE
# =============================================================================

banner("STAGE26-8D3 :: BUILD T26_PARETO")

T7 = []

for src in pareto_rows:
    target = src["target_id"]
    if target not in {
        "STAGE16_XGBOOST_TUNED",
        "STAGE16_LIGHTGBM_TUNED",
        "STAGE16_CATBOOST_TUNED",
        "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
        "STAGE20_MASKED_CNN_V1",
        "STAGE21_MASKED_VIT_V1",
    }:
        raise RuntimeError(f"Unexpected Pareto target: {target}")

    member = parse_bool(src["frontier_member"], "Pareto frontier_member")

    T7.append(
        {
            "group": src["comparison_group"],
            "target_id": target,
            "pr_auc": src["pr_auc"],
            "cpu1_batch1_p95_latency_ms": (
                src["p95_cpu1_batch1_inference_latency_ms"]
            ),
            "pareto_status": (
                "FRONTIER_MEMBER" if member else "NOT_FRONTIER_MEMBER"
            ),
        }
    )

T7.sort(key=lambda row: target_sort_key(row["target_id"]))

if len(T7) != 6:
    raise RuntimeError("T26_PARETO must contain exactly six primary-architecture rows.")


# =============================================================================
# 14. T8 — PAIRED REPRESENTATION IMPLEMENTATION SENSITIVITY
# =============================================================================

banner("STAGE26-8D3 :: BUILD T26_REPRESENTATION_SENSITIVITY")

if rep_audit.get("status") != (
    "PASS_PAIRED_4C3_IMPLEMENTATION_SENSITIVITY_AUDIT_COMPLETED"
):
    raise RuntimeError("Unexpected Stage26-8 representation-sensitivity audit status.")

if rep_audit["statistical_policy"].get("bootstrap") is not False:
    raise RuntimeError("Stage26-8 sensitivity unexpectedly permits bootstrap.")
if rep_audit["statistical_policy"].get("hypothesis_test") is not False:
    raise RuntimeError("Stage26-8 sensitivity unexpectedly permits hypothesis testing.")
if rep_audit["statistical_policy"].get("materiality_threshold") is not None:
    raise RuntimeError("Stage26-8 sensitivity unexpectedly has a materiality threshold.")
if rep_audit["execution_geometry"].get("all_pair_fingerprints_equal") is not True:
    raise RuntimeError("Stage26-8 audit does not certify equal pair fingerprints.")

supported_interpretation = (
    rep_audit["descriptive_interpretation"]["supported_statement"]
)
audit_batches = rep_audit["descriptive_batch_results"]

pairs_by_batch = defaultdict(list)
for pair in rep_pair_rows:
    batch = parse_int(pair["batch_size"], "sensitivity pair batch")
    pairs_by_batch[batch].append(pair)

T8 = []

for src in rep_batch_rows:
    batch = parse_int(src["batch_size"], "sensitivity summary batch")
    if batch not in BATCH_RANK:
        raise RuntimeError(f"Unexpected sensitivity batch: {batch}")

    if parse_bool(src["bootstrap"], "sensitivity bootstrap"):
        raise RuntimeError("Sensitivity batch row unexpectedly indicates bootstrap.")
    if parse_bool(src["hypothesis_test"], "sensitivity hypothesis_test"):
        raise RuntimeError("Sensitivity batch row unexpectedly indicates a hypothesis test.")
    if str(src["materiality_threshold"]).strip() != "":
        raise RuntimeError("Sensitivity batch row unexpectedly has a materiality threshold.")

    n_pairs = parse_int(src["complete_pair_count"], "complete_pair_count")
    batch_pairs = pairs_by_batch[batch]
    complete_pairs = [
        pair
        for pair in batch_pairs
        if parse_bool(pair["pair_complete"], "pair_complete")
    ]

    if len(complete_pairs) != n_pairs:
        raise RuntimeError(
            f"Sensitivity complete-pair count mismatch for batch {batch}."
        )

    fingerprint_equal_count = sum(
        1
        for pair in complete_pairs
        if parse_bool(pair["fingerprint_equal"], "fingerprint_equal")
    )

    if fingerprint_equal_count != n_pairs:
        raise RuntimeError(
            f"Sensitivity fingerprint equality failed for batch {batch}."
        )

    audit_row = audit_batches.get(str(batch))
    if audit_row is None:
        raise RuntimeError(f"Sensitivity audit lacks batch {batch}.")

    if not numeric_equal(
        src["throughput_ratio_median"],
        audit_row["throughput_ratio_V3_over_V2_median"],
        f"sensitivity throughput median batch {batch}",
    ):
        raise RuntimeError(
            f"Sensitivity batch-summary/audit throughput median mismatch: batch {batch}."
        )

    T8.append(
        {
            "batch_size": str(batch),
            "n_pairs": str(n_pairs),
            "fingerprints_equal_count": str(fingerprint_equal_count),
            "median_throughput_ratio_v3_over_v2": src["throughput_ratio_median"],
            "effect_direction": audit_row["throughput_direction"],
            "interpretation": supported_interpretation,
        }
    )

T8.sort(key=lambda row: batch_sort_key(row["batch_size"]))

if len(T8) != 5:
    raise RuntimeError("T26_REPRESENTATION_SENSITIVITY must contain five rows.")


# =============================================================================
# 15. WRITE TABLES USING EXACT EFFECTIVE-SCHEMA COLUMN ORDER
# =============================================================================

banner("STAGE26-8D3 :: WRITE EIGHT TABLES")

OUT_DIR.mkdir(parents=True, exist_ok=False)

TABLE_DATA = {
    "T26_CPU_WARM_INFERENCE": T1,
    "T26_CPU_MEMORY_PACKAGE": T2,
    "T26_CPU_COLD_START": T3,
    "T26_CPU_CAPACITY_SCALING": T4,
    "T26_COMPONENT_MEASUREMENTS": T5,
    "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO": T6,
    "T26_PARETO": T7,
    "T26_REPRESENTATION_SENSITIVITY": T8,
}

output_records = []

for table_id in expected_table_ids:
    spec = table_spec(schema, table_id)
    columns = spec["columns"]
    rows = TABLE_DATA[table_id]
    path = TABLE_PATHS[table_id]

    atomic_csv(path, columns, rows)

    # Re-open and verify exact header + row count.
    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        actual_header = next(reader)
        actual_row_count = sum(1 for _ in reader)

    if actual_header != columns:
        raise RuntimeError(f"Output header differs from effective schema: {table_id}")
    if actual_row_count != len(rows):
        raise RuntimeError(f"Output row count mismatch: {table_id}")

    record = {
        "table_id": table_id,
        "title": spec["title"],
        "repo_relative_path": str(path.relative_to(REPO)),
        "columns": columns,
        "row_count": len(rows),
        "sha256": sha256_file(path),
        "size_bytes": int(path.stat().st_size),
    }
    output_records.append(record)

    print(f"{table_id:50s} rows={len(rows):4d} SHA256={record['sha256']}")


# =============================================================================
# 16. PUBLICATION-SAFETY VALIDATION
# =============================================================================

banner("STAGE26-8D3 :: PUBLICATION-SAFETY VALIDATION")

# Cold-start table must never expose stored Stage26-1 CIs.
cold_columns = table_spec(schema, "T26_CPU_COLD_START")["columns"]
for forbidden in [
    "p50_bootstrap_ci95_low_ms",
    "p50_bootstrap_ci95_high_ms",
    "p95_bootstrap_ci95_low_ms",
    "p95_bootstrap_ci95_high_ms",
]:
    if forbidden in cold_columns:
        raise RuntimeError(f"Forbidden cold-start CI column survived: {forbidden}")

# Warm table must not contain historical Stage26-2 CI names.
warm_columns = table_spec(schema, "T26_CPU_WARM_INFERENCE")["columns"]
if any("historical" in column.lower() for column in warm_columns):
    raise RuntimeError("Historical Stage26-2 uncertainty leaked into warm publication table.")

# T4 and T6 values are copied, not recomputed.
if table_spec(schema, "T26_CPU_CAPACITY_SCALING")["rules"][0] != (
    "Copy immutable Stage26-7C descriptive values; do not recompute them."
):
    raise RuntimeError("Effective T4 immutable-copy rule changed.")

if table_spec(schema, "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO")[
    "frozen_formula"
] != (
    "inference_median_throughput_flows_per_second / "
    "representation_median_flows_per_second"
):
    raise RuntimeError("Effective T6 frozen formula changed.")

# T5 must explicitly retain complete E2E unavailability.
if any(row["complete_e2e_available"] is not False for row in T5):
    raise RuntimeError("T5 contains an invalid complete-E2E availability claim.")

# Pareto remains within-group only.
pareto_groups = {row["group"] for row in T7}
if pareto_groups != {"GROUP_A_DUPSAFE70", "GROUP_B_PACKET_IMAGE"}:
    raise RuntimeError("Unexpected Pareto group universe.")

print("Cold-start CIs absent                : PASS")
print("Historical Stage26-2 CIs absent      : PASS")
print("Stage26-7C ratios/scaling not redone : PASS")
print("Group-B ratio not inverted           : PASS")
print("Memory condition aggregation         : NONE")
print("Complete E2E                         : UNAVAILABLE")
print("Pareto recomputation                 : NONE")
print("Stage26-8 inference/bootstrap        : NONE")
print("GPU                                  : OFF")


# =============================================================================
# 17. WRITE INDEX / RECEIPT / MANIFEST
# =============================================================================

banner("STAGE26-8D3 :: WRITE INDEX / RECEIPT / MANIFEST")

index_payload = {
    "schema": "stage26_8d3_cpu_publication_tables_index_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "scientific_parent": EXPECTED_PARENT,
    "effective_schema_repo_relative_path": str(EFFECTIVE_SCHEMA.relative_to(REPO)),
    "effective_schema_sha256": EXPECTED_EFFECTIVE_SCHEMA_SHA256,
    "publication_label": PUBLICATION_LABEL,
    "table_count": len(output_records),
    "tables": output_records,
}
atomic_json(INDEX_PATH, index_payload)
index_sha = sha256_file(INDEX_PATH)

receipt_payload = {
    "schema": "stage26_8d3_cpu_publication_tables_receipt_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "scientific_parent": EXPECTED_PARENT,
    "effective_schema_sha256": EXPECTED_EFFECTIVE_SCHEMA_SHA256,
    "source_identity": source_identity,
    "table_count": len(output_records),
    "table_row_counts": {
        record["table_id"]: record["row_count"] for record in output_records
    },
    "deterministic_publication_transforms": [
        "SOURCE_COLUMN_SELECTION_AND_RENAMING",
        "EXACT_KEY_JOINS",
        "BYTES_TO_MIB_UNIT_CONVERSION",
        "MEMORY_STATUS_LABEL_FROM_FROZEN_REPETITION_COUNTS",
        "PAIR_FINGERPRINT_EQUALITY_COUNT_REQUIRED_BY_FROZEN_T8_SCHEMA",
    ],
    "scientific_safety": {
        "measured_values_changed": False,
        "timing_executed": False,
        "inference_executed": False,
        "model_loaded": False,
        "bootstrap_executed": False,
        "random_sampling_executed": False,
        "cold_start_ci_used": False,
        "historical_stage26_2_ci_used": False,
        "memory_aggregated_across_cpu_or_batch": False,
        "stage26_7c_recomputed": False,
        "group_b_ratio_recomputed": False,
        "group_b_ratio_inverted": False,
        "pareto_recomputed": False,
        "stage26_8_hypothesis_test": False,
        "stage26_8_materiality_threshold": False,
        "pcap_accessed": False,
        "labels_accessed": False,
        "gpu_used": False,
    },
    "complete_e2e": "UNAVAILABLE",
    "historical_stage26_4c3": "RETAINED_AS_ORIGINAL_MEASUREMENT",
    "stage26_8_representation_sensitivity": "DESCRIPTIVE_SEPARATE_NOT_REPLACEMENT",
}
atomic_json(RECEIPT_PATH, receipt_payload)
receipt_sha = sha256_file(RECEIPT_PATH)

manifest_files = []
for path in [*TABLE_PATHS.values(), INDEX_PATH, RECEIPT_PATH]:
    manifest_files.append(
        {
            "path": str(path.relative_to(REPO)),
            "sha256": sha256_file(path),
            "size_bytes": int(path.stat().st_size),
        }
    )

manifest_payload = {
    "schema": "stage26_8d3_cpu_publication_tables_manifest_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "scientific_parent": EXPECTED_PARENT,
    "files": manifest_files,
}
atomic_json(MANIFEST_PATH, manifest_payload)
manifest_sha = sha256_file(MANIFEST_PATH)

print("Index SHA256   :", index_sha)
print("Receipt SHA256 :", receipt_sha)
print("Manifest SHA256:", manifest_sha)


# =============================================================================
# 18. PRE-COMMIT GIT AUDIT
# =============================================================================

banner("STAGE26-8D3 :: PRE-COMMIT GIT AUDIT")

status_lines = [
    line for line in git("status", "--porcelain").splitlines() if line.strip()
]

for line in status_lines:
    print(line)

expected_prefix = "?? " + str(OUT_DIR.relative_to(REPO))

if not status_lines:
    raise RuntimeError("No Stage26-8D3 files detected by Git.")
if any(not line.startswith(expected_prefix) for line in status_lines):
    raise RuntimeError(
        "Unexpected repository modification detected before Stage26-8D3 commit."
    )


# =============================================================================
# 19. COMMIT
# =============================================================================

banner("STAGE26-8D3 :: COMMIT")

git("add", str(OUT_DIR.relative_to(REPO)))

staged = git("diff", "--cached", "--name-only").splitlines()
expected_staged = sorted(
    [
        *(str(path.relative_to(REPO)) for path in TABLE_PATHS.values()),
        str(INDEX_PATH.relative_to(REPO)),
        str(RECEIPT_PATH.relative_to(REPO)),
        str(MANIFEST_PATH.relative_to(REPO)),
    ]
)

print("Staged files:")
for path in staged:
    print(" ", path)

if sorted(staged) != expected_staged:
    raise RuntimeError(
        "Stage26-8D3 staged-file set differs from the expected 11 files."
    )

git("commit", "-m", COMMIT_MESSAGE)

new_head = git("rev-parse", "HEAD")
parent = git("rev-parse", "HEAD^")
subject = git("log", "-1", "--pretty=%s")

print("Parent :", parent)
print("HEAD   :", new_head)
print("Subject:", subject)

if parent != EXPECTED_PARENT:
    raise RuntimeError("Stage26-8D3 commit parent mismatch.")
if subject != COMMIT_MESSAGE:
    raise RuntimeError("Unexpected Stage26-8D3 commit subject.")


# =============================================================================
# 20. AUTHENTICATED PUSH
# =============================================================================

banner("STAGE26-8D3 :: PUSH")

try:
    from kaggle_secrets import UserSecretsClient
except Exception as exc:
    raise RuntimeError("Kaggle UserSecretsClient unavailable.") from exc

try:
    github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "Could not read Kaggle Secret GITHUB_TOKEN. "
        "Do not paste the token into notebook source."
    ) from exc

if not github_token or len(github_token.strip()) < 20:
    raise RuntimeError("GITHUB_TOKEN secret is empty or unusable.")

askpass_fd, askpass_name = tempfile.mkstemp(
    prefix="stage26_8d3_askpass_",
    suffix=".sh",
)
os.close(askpass_fd)
askpass_path = Path(askpass_name)

try:
    askpass_path.write_text(
        "#!/bin/sh\n"
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *Password*) printf "%s\\n" "$STAGE26_GITHUB_TOKEN" ;;\n'
        '  *) printf "%s\\n" "" ;;\n'
        "esac\n",
        encoding="utf-8",
    )
    askpass_path.chmod(
        askpass_path.stat().st_mode
        | stat.S_IXUSR
        | stat.S_IXGRP
        | stat.S_IXOTH
    )

    push_env = os.environ.copy()
    push_env["GIT_ASKPASS"] = str(askpass_path)
    push_env["GIT_TERMINAL_PROMPT"] = "0"
    push_env["STAGE26_GITHUB_TOKEN"] = github_token.strip()

    push_output = git("push", "origin", "main", env=push_env)
    print(push_output if push_output else "<push completed>")

finally:
    try:
        askpass_path.unlink(missing_ok=True)
    except Exception:
        pass
    github_token = None
    if "push_env" in locals():
        push_env.pop("STAGE26_GITHUB_TOKEN", None)


# =============================================================================
# 21. REMOTE BYTE VERIFICATION
# =============================================================================

banner("STAGE26-8D3 :: REMOTE BYTE VERIFICATION")

git("fetch", "origin", "main")
local_after = git("rev-parse", "HEAD")
remote_after = git("rev-parse", "origin/main")

print("Local HEAD :", local_after)
print("origin/main:", remote_after)

if local_after != new_head or remote_after != new_head:
    raise RuntimeError("Local/remote commit mismatch after Stage26-8D3 push.")

all_output_paths = [
    *TABLE_PATHS.values(),
    INDEX_PATH,
    RECEIPT_PATH,
    MANIFEST_PATH,
]

verification = []

for path in all_output_paths:
    rel = str(path.relative_to(REPO))
    local_bytes = path.read_bytes()
    remote_bytes = git_blob_bytes("origin/main", rel)
    local_sha = sha256_bytes(local_bytes)
    remote_sha = sha256_bytes(remote_bytes)
    same = local_bytes == remote_bytes
    verification.append(
        {
            "path": rel,
            "local_sha256": local_sha,
            "remote_sha256": remote_sha,
            "byte_identical": same,
        }
    )
    print(f"{'PASS' if same else 'FAIL'} {rel}")
    print("  local :", local_sha)
    print("  remote:", remote_sha)
    if not same:
        raise RuntimeError(f"Remote byte verification failed: {rel}")


# =============================================================================
# 22. FINAL CLEAN-REPO CLOSURE
# =============================================================================

final_status = git("status", "--porcelain")

banner("STAGE26-8D3 COMPLETE")

print("Durable table-generation anchor:", new_head)
print("HEAD == origin/main          :", local_after == remote_after == new_head)
print("Repo clean                   :", final_status == "")
print(
    "Remote files byte-identical  :",
    all(item["byte_identical"] for item in verification),
)
print("Publication tables generated :", len(output_records))

print("\nROW COUNTS:")
for record in output_records:
    print(f"  {record['table_id']:50s} {record['row_count']}")

print("\nSCIENTIFIC STATE:")
print("  effective schema            : STAGE26-8D2 + STAGE26-8D2A")
print("  cold-start uncertainty      : POINT ESTIMATES ONLY")
print("  warm uncertainty            : STAGE26-6F1 CORRECTED ONLY")
print("  memory aggregation          : NONE")
print("  Stage26-7C                  : COPIED / NOT RECOMPUTED")
print("  Group-B ratio               : COPIED / NOT INVERTED")
print("  Stage26-6D Pareto           : COPIED / NOT RECOMPUTED")
print("  historical Stage26-4C3      : RETAINED")
print("  Stage26-8 sensitivity       : DESCRIPTIVE / SEPARATE")
print("  complete E2E                : UNAVAILABLE")
print("  timing                      : NO")
print("  inference                   : NO")
print("  bootstrap                   : NO")
print("  PCAP                        : NO")
print("  labels                      : NO")
print("  GPU                         : OFF")

if final_status:
    raise RuntimeError("Repository is not clean after Stage26-8D3.")

print("\nNEXT:")
print("  Generate the eight CPU publication figures from the frozen schema")
print("  and the now-durable Stage26-8D3 publication tables.")
print("  GPU remains OFF.")


In [18]:
# =============================================================================
# STAGE26-8D3R1
# WARM-INFERENCE NON-PASS JOIN DIAGNOSTIC
#
# READ-ONLY RECOVERY CELL
#
# PURPOSE
# -------
# Confirm that the Stage26-8D3 failure was caused solely by the known
# Stage26-6F1 representation of non-PASS rows:
#
#   - condition_id retained
#   - resource-limit status retained
#   - identity fields intentionally blank
#   - corrected CI status = NOT_APPLICABLE_NON_PASS
#
# This cell DOES NOT:
#   - generate publication tables
#   - write files
#   - modify Git
#   - run timing
#   - run inference
#   - load models
#   - run bootstrap
#   - recompute Stage26-7C
#   - recompute Pareto
#   - access PCAP/labels
#   - use GPU
# =============================================================================

from __future__ import annotations

import csv
import hashlib
import subprocess
from pathlib import Path


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

RESULT_ROOT = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
)

EXPECTED_HEAD = (
    "80f7fce76b3a859ea699e8838eced34e9936d421"
)

EXPECTED_EFFECTIVE_SCHEMA_SHA256 = (
    "973cc58ed6572d30b1cce94f3659226d99b185390c2aca92d9ed030e50358ad0"
)

EFFECTIVE_SCHEMA = (
    RESULT_ROOT
    / "stage26_8d2a_cpu_publication_schema_erratum"
    / "stage26_8d2a_effective_cpu_publication_schema.json"
)

WARM_STATUS_CSV = (
    RESULT_ROOT
    / "stage26_2_cpu_warm_inference"
    / "stage26_2_condition_status.csv"
)

WARM_6F1_CSV = (
    RESULT_ROOT
    / "stage26_6f1_bootstrap_corrected_uncertainty"
    / "stage26_6f1_corrected_timing_uncertainty.csv"
)

OUT_DIR = (
    RESULT_ROOT
    / "stage26_8d3_cpu_publication_tables"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):
    print("\n" + "=" * 118)
    print(text)
    print("=" * 118)


def run(cmd, check=True):
    p = subprocess.run(
        cmd,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(cmd)}\n"
            f"{p.stdout}"
        )

    return p.stdout.strip()


def git(*args):
    return run(["git", *args])


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(8 * 1024 * 1024)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def read_csv(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:
        return list(csv.DictReader(f))


def index_unique(rows, key, label):
    out = {}

    for row in rows:
        value = row[key]

        if value in out:
            raise RuntimeError(
                f"Duplicate {label} key: {value}"
            )

        out[value] = row

    return out


# =============================================================================
# 2. DURABLE STATE / NO-PARTIAL-OUTPUT GATE
# =============================================================================

banner(
    "STAGE26-8D3R1 :: DURABLE STATE GATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)

print(
    "Expected HEAD :",
    EXPECTED_HEAD,
)

print(
    "Local HEAD    :",
    head,
)

print(
    "origin/main   :",
    origin,
)

print(
    "Repo clean    :",
    status == "",
)

print(
    "8D3 OUT_DIR exists:",
    OUT_DIR.exists(),
)


if head != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected local HEAD."
    )


if origin != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected origin/main."
    )


if status:
    raise RuntimeError(
        "Repository is not clean after failed Stage26-8D3."
    )


if OUT_DIR.exists():
    raise RuntimeError(
        "Stage26-8D3 output directory exists unexpectedly. "
        "Do not rerun the generator until this is inspected."
    )


# =============================================================================
# 3. EFFECTIVE-SCHEMA IDENTITY
# =============================================================================

banner(
    "STAGE26-8D3R1 :: EFFECTIVE SCHEMA IDENTITY"
)

if not EFFECTIVE_SCHEMA.is_file():
    raise FileNotFoundError(
        EFFECTIVE_SCHEMA
    )

schema_sha = sha256_file(
    EFFECTIVE_SCHEMA
)

print(
    "Expected SHA256:",
    EXPECTED_EFFECTIVE_SCHEMA_SHA256,
)

print(
    "Actual SHA256  :",
    schema_sha,
)


if schema_sha != EXPECTED_EFFECTIVE_SCHEMA_SHA256:
    raise RuntimeError(
        "Effective Stage26-8D2A schema SHA mismatch."
    )


# =============================================================================
# 4. LOAD ONLY EXISTING COMMITTED TABLE SOURCES
# =============================================================================

banner(
    "STAGE26-8D3R1 :: LOAD WARM SOURCE TABLES"
)

if not WARM_STATUS_CSV.is_file():
    raise FileNotFoundError(
        WARM_STATUS_CSV
    )


if not WARM_6F1_CSV.is_file():
    raise FileNotFoundError(
        WARM_6F1_CSV
    )


stage2_rows = read_csv(
    WARM_STATUS_CSV
)

stage6f1_rows = read_csv(
    WARM_6F1_CSV
)


print(
    "Stage26-2 rows :",
    len(stage2_rows),
)

print(
    "Stage26-6F1 rows:",
    len(stage6f1_rows),
)


if len(stage2_rows) != 80:
    raise RuntimeError(
        "Expected 80 Stage26-2 conditions."
    )


if len(stage6f1_rows) != 80:
    raise RuntimeError(
        "Expected 80 Stage26-6F1 conditions."
    )


stage2 = index_unique(
    stage2_rows,
    "condition_id",
    "Stage26-2 condition",
)

stage6f1 = index_unique(
    stage6f1_rows,
    "condition_id",
    "Stage26-6F1 condition",
)


if set(stage2) != set(stage6f1):
    raise RuntimeError(
        "Stage26-2 and Stage26-6F1 condition-ID sets differ."
    )


# =============================================================================
# 5. CLASSIFY PASS / NON-PASS
# =============================================================================

banner(
    "STAGE26-8D3R1 :: PASS / NON-PASS STRUCTURE"
)

stage2_pass = [
    row
    for row in stage2_rows
    if row["status"] == "PASS"
]

stage2_nonpass = [
    row
    for row in stage2_rows
    if row["status"] != "PASS"
]


print(
    "PASS conditions    :",
    len(stage2_pass),
)

print(
    "Non-PASS conditions:",
    len(stage2_nonpass),
)


if len(stage2_pass) != 73:
    raise RuntimeError(
        "Expected 73 PASS warm conditions."
    )


if len(stage2_nonpass) != 7:
    raise RuntimeError(
        "Expected 7 warm resource-limit conditions."
    )


status_counts = {}

for row in stage2_nonpass:
    status_counts[
        row["status"]
    ] = (
        status_counts.get(
            row["status"],
            0,
        )
        + 1
    )


print(
    "Non-PASS status counts:",
    status_counts,
)


expected_status_counts = {
    "RESOURCE_LIMIT_OOM": 2,
    "TIMEOUT_RESOURCE_LIMIT": 5,
}


if status_counts != expected_status_counts:
    raise RuntimeError(
        "Warm non-PASS outcome counts changed."
    )


# =============================================================================
# 6. VERIFY ALL PASS ROWS HAVE FULL IDENTITY IN 6F1
# =============================================================================

banner(
    "STAGE26-8D3R1 :: PASS-ROW IDENTITY CHECK"
)

identity_fields = [
    "target_id",
    "comparison_group",
    "hardware_mode",
    "thread_count",
    "batch_size",
    "status",
]


for src in stage2_pass:
    cid = src[
        "condition_id"
    ]

    unc = stage6f1[
        cid
    ]

    for field in identity_fields:
        if src[field] != unc[field]:
            raise RuntimeError(
                f"PASS identity mismatch for {cid}: "
                f"{field}={src[field]!r} vs {unc[field]!r}"
            )


    if (
        src["timed_runs_observed"]
        !=
        unc["n"]
    ):
        raise RuntimeError(
            f"PASS observation-count mismatch for {cid}: "
            f"{src['timed_runs_observed']!r} vs {unc['n']!r}"
        )


    if (
        unc["corrected_ci_status"]
        !=
        "PROTOCOL_CERTIFIED_DERIVED_UNCERTAINTY"
    ):
        raise RuntimeError(
            f"Unexpected corrected CI certification for {cid}: "
            f"{unc['corrected_ci_status']!r}"
        )


print(
    "PASS identity rows checked:",
    len(stage2_pass),
)

print(
    "PASS identity result      : PASS"
)


# =============================================================================
# 7. VERIFY NON-PASS 6F1 REPRESENTATION
# =============================================================================

banner(
    "STAGE26-8D3R1 :: NON-PASS REPRESENTATION CHECK"
)

blank_identity_fields = [
    "target_id",
    "comparison_group",
    "hardware_mode",
    "thread_count",
    "batch_size",
]


nonpass_report = []


for src in sorted(
    stage2_nonpass,
    key=lambda row: row[
        "condition_id"
    ],
):

    cid = src[
        "condition_id"
    ]

    unc = stage6f1[
        cid
    ]


    if unc[
        "status"
    ] != src[
        "status"
    ]:
        raise RuntimeError(
            f"Non-PASS status mismatch for {cid}: "
            f"Stage26-2={src['status']!r}, "
            f"Stage26-6F1={unc['status']!r}"
        )


    nonblank = {
        field: unc[field]
        for field in blank_identity_fields
        if str(
            unc[field]
        ).strip()
        != ""
    }


    if nonblank:
        raise RuntimeError(
            f"Expected blank Stage26-6F1 non-PASS identity fields "
            f"for {cid}; found {nonblank}."
        )


    if str(
        unc[
            "n"
        ]
    ).strip() != "":
        raise RuntimeError(
            f"Expected blank Stage26-6F1 n for non-PASS {cid}."
        )


    if (
        unc[
            "historical_ci_status"
        ]
        !=
        "NOT_APPLICABLE_NON_PASS"
    ):
        raise RuntimeError(
            f"Unexpected historical CI status for non-PASS {cid}: "
            f"{unc['historical_ci_status']!r}"
        )


    if (
        unc[
            "corrected_ci_status"
        ]
        !=
        "NOT_APPLICABLE_NON_PASS"
    ):
        raise RuntimeError(
            f"Unexpected corrected CI status for non-PASS {cid}: "
            f"{unc['corrected_ci_status']!r}"
        )


    nonpass_report.append(
        (
            cid,
            src[
                "target_id"
            ],
            src[
                "hardware_mode"
            ],
            src[
                "batch_size"
            ],
            src[
                "status"
            ],
        )
    )


print(
    "Condition     Target                                      "
    "Hardware                 Batch   Status"
)

print(
    "-" * 118
)


for (
    cid,
    target,
    hardware,
    batch,
    outcome,
) in nonpass_report:

    print(
        f"{cid:12s} "
        f"{target:43s} "
        f"{hardware:24s} "
        f"{batch:7s} "
        f"{outcome}"
    )


print(
    "\nNon-PASS Stage26-6F1 identity fields: "
    "INTENTIONALLY BLANK"
)

print(
    "Non-PASS Stage26-6F1 CI status      : "
    "NOT_APPLICABLE_NON_PASS"
)


# =============================================================================
# 8. EXPLICITLY INSPECT CPUCOND_050
# =============================================================================

banner(
    "STAGE26-8D3R1 :: CPUCOND_050 ROOT-CAUSE CONFIRMATION"
)

src_050 = stage2[
    "CPUCOND_050"
]

unc_050 = stage6f1[
    "CPUCOND_050"
]


print(
    "Stage26-2:"
)

for field in [
    "condition_id",
    "target_id",
    "comparison_group",
    "hardware_mode",
    "thread_count",
    "batch_size",
    "status",
]:
    print(
        f"  {field:20s}: "
        f"{src_050[field]!r}"
    )


print(
    "\nStage26-6F1:"
)

for field in [
    "condition_id",
    "target_id",
    "comparison_group",
    "hardware_mode",
    "thread_count",
    "batch_size",
    "status",
    "n",
    "corrected_ci_status",
]:
    print(
        f"  {field:20s}: "
        f"{unc_050[field]!r}"
    )


if (
    src_050[
        "target_id"
    ]
    !=
    "STAGE20_MASKED_CNN_V1"
):
    raise RuntimeError(
        "Unexpected CPUCOND_050 Stage26-2 target."
    )


if (
    src_050[
        "status"
    ]
    !=
    "RESOURCE_LIMIT_OOM"
):
    raise RuntimeError(
        "Unexpected CPUCOND_050 Stage26-2 status."
    )


if str(
    unc_050[
        "target_id"
    ]
).strip() != "":
    raise RuntimeError(
        "CPUCOND_050 6F1 target_id was expected to be blank."
    )


if (
    unc_050[
        "status"
    ]
    !=
    "RESOURCE_LIMIT_OOM"
):
    raise RuntimeError(
        "Unexpected CPUCOND_050 6F1 status."
    )


# =============================================================================
# 9. FINAL RECOVERY VERDICT
# =============================================================================

banner(
    "STAGE26-8D3R1 COMPLETE"
)


print(
    "Repository modified           : NO"
)

print(
    "Publication artifacts written : NO"
)

print(
    "Timing executed               : NO"
)

print(
    "Inference executed            : NO"
)

print(
    "Model loaded                  : NO"
)

print(
    "Bootstrap executed            : NO"
)

print(
    "Stage26-7C recomputed         : NO"
)

print(
    "Pareto recomputed             : NO"
)

print(
    "PCAP accessed                 : NO"
)

print(
    "Labels accessed               : NO"
)

print(
    "GPU                           : OFF"
)


print(
    "\nROOT CAUSE:"
)

print(
    "  Stage26-8D3 incorrectly required Stage26-6F1 "
    "identity fields on non-PASS rows."
)

print(
    "  Stage26-6F1 intentionally leaves those identity fields "
    "blank because uncertainty is NOT_APPLICABLE_NON_PASS."
)


print(
    "\nCORRECT GENERATOR POLICY:"
)

print(
    "  PASS rows:"
)

print(
    "    - require full Stage26-2 / Stage26-6F1 identity equality"
)

print(
    "    - require exact point-estimate equality"
)

print(
    "    - use Stage26-6F1 corrected CIs"
)

print(
    "  NON-PASS rows:"
)

print(
    "    - preserve target/group/hardware/batch identity from Stage26-2"
)

print(
    "    - require Stage26-6F1 condition_id + status agreement only"
)

print(
    "    - require NOT_APPLICABLE_NON_PASS"
)

print(
    "    - leave all timing/CI publication fields unavailable"
)

print(
    "    - preserve OOM/timeout exactly"
)


print(
    "\nRECOVERY_DECISION:"
)

print(
    "PATCH_STAGE26_8D3_PASS_AWARE_6F1_JOIN"
)


STAGE26-8D3R1 :: DURABLE STATE GATE
Expected HEAD : 80f7fce76b3a859ea699e8838eced34e9936d421
Local HEAD    : 80f7fce76b3a859ea699e8838eced34e9936d421
origin/main   : 80f7fce76b3a859ea699e8838eced34e9936d421
Repo clean    : True
8D3 OUT_DIR exists: False

STAGE26-8D3R1 :: EFFECTIVE SCHEMA IDENTITY
Expected SHA256: 973cc58ed6572d30b1cce94f3659226d99b185390c2aca92d9ed030e50358ad0
Actual SHA256  : 973cc58ed6572d30b1cce94f3659226d99b185390c2aca92d9ed030e50358ad0

STAGE26-8D3R1 :: LOAD WARM SOURCE TABLES
Stage26-2 rows : 80
Stage26-6F1 rows: 80

STAGE26-8D3R1 :: PASS / NON-PASS STRUCTURE
PASS conditions    : 73
Non-PASS conditions: 7
Non-PASS status counts: {'RESOURCE_LIMIT_OOM': 2, 'TIMEOUT_RESOURCE_LIMIT': 5}

STAGE26-8D3R1 :: PASS-ROW IDENTITY CHECK
PASS identity rows checked: 73
PASS identity result      : PASS

STAGE26-8D3R1 :: NON-PASS REPRESENTATION CHECK
Condition     Target                                      Hardware                 Batch   Status
--------------------------------

In [19]:
# =============================================================================
# STAGE26-8D3
# GENERATE EIGHT CPU PUBLICATION TABLES FROM THE EFFECTIVE 8D2+8D2A SCHEMA
#
# SCIENTIFIC PARENT:
#   80f7fce76b3a859ea699e8838eced34e9936d421
#
# PURPOSE
# -------
# Materialize the eight frozen CPU publication tables using only already-
# committed Stage26 artifacts and the effective Stage26-8D2A schema.
#
# This cell performs deterministic publication-layer transforms only:
#   - source-column selection / renaming
#   - exact-key joins
#   - bytes -> MiB unit conversion for frozen Stage26-3B memory metrics
#   - representation-only status labels from frozen repetition counts
#   - fingerprint-equality counting required by the frozen sensitivity table
#
# It DOES NOT:
#   - run timing
#   - run inference or load models
#   - bootstrap
#   - resample or use randomness
#   - recompute Stage26-7C ratios/scaling
#   - invert Group-B ratios
#   - recompute Pareto membership
#   - aggregate memory across CPU modes or batch sizes
#   - use cold-start confidence intervals
#   - use historical Stage26-2 confidence intervals
#   - access PCAP or labels
#   - use GPU
#
# OUTPUT
# ------
# results/stage26_deployment_profiling/stage26_8d3_cpu_publication_tables/
#   T26_CPU_WARM_INFERENCE.csv
#   T26_CPU_MEMORY_PACKAGE.csv
#   T26_CPU_COLD_START.csv
#   T26_CPU_CAPACITY_SCALING.csv
#   T26_COMPONENT_MEASUREMENTS.csv
#   T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO.csv
#   T26_PARETO.csv
#   T26_REPRESENTATION_SENSITIVITY.csv
#   stage26_8d3_cpu_publication_tables_index.json
#   stage26_8d3_cpu_publication_tables_receipt.json
#   stage26_8d3_cpu_publication_tables_manifest.json
# =============================================================================

from __future__ import annotations

import csv
import hashlib
import json
import os
import stat
import subprocess
import tempfile
from collections import defaultdict
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from pathlib import Path


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")
RESULT_ROOT = REPO / "results" / "stage26_deployment_profiling"

EXPECTED_PARENT = "80f7fce76b3a859ea699e8838eced34e9936d421"

EFFECTIVE_SCHEMA = (
    RESULT_ROOT
    / "stage26_8d2a_cpu_publication_schema_erratum"
    / "stage26_8d2a_effective_cpu_publication_schema.json"
)
EXPECTED_EFFECTIVE_SCHEMA_SHA256 = (
    "973cc58ed6572d30b1cce94f3659226d99b185390c2aca92d9ed030e50358ad0"
)

PUBLICATION_LABEL = (
    "COMPONENT_LEVEL_WITH_STAGE26_8_4C3_SENSITIVITY_AUDIT_COMPLETED"
)

TARGET_ORDER = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
    "ENS_LGBM_XGB_EQUAL",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
]

TARGET_RANK = {target: i for i, target in enumerate(TARGET_ORDER)}
CPU_ORDER = {
    "CPU_1_PHYSICAL_CORE": 0,
    "CPU_2_PHYSICAL_CORE": 1,
}
BATCH_ORDER = [1, 64, 256, 1024, 8192]
BATCH_RANK = {batch: i for i, batch in enumerate(BATCH_ORDER)}

COLD_COMPONENT_ORDER = [
    "process_spawn_to_worker_ready_ns",
    "framework_import_ns",
    "input_preparation_ns",
    "model_deserialization_load_ns",
    "first_prediction_ns",
    "spawn_to_first_output_ns",
    "parent_process_total_ns",
]
COLD_COMPONENT_RANK = {
    component: i for i, component in enumerate(COLD_COMPONENT_ORDER)
}

# -----------------------------------------------------------------------------
# Source files
# -----------------------------------------------------------------------------

WARM_STATUS = (
    RESULT_ROOT
    / "stage26_2_cpu_warm_inference"
    / "stage26_2_condition_status.csv"
)

WARM_CORRECTED_UNCERTAINTY = (
    RESULT_ROOT
    / "stage26_6f1_bootstrap_corrected_uncertainty"
    / "stage26_6f1_corrected_timing_uncertainty.csv"
)

COLD_IMPLEMENTATION = (
    RESULT_ROOT
    / "stage26_1_cpu_cold_start"
    / "stage26_1_cold_start_implementation.json"
)

COLD_SUMMARY = (
    RESULT_ROOT
    / "stage26_1_cpu_cold_start"
    / "stage26_1_cold_start_summary.csv"
)

MEMORY_SUMMARY = (
    RESULT_ROOT
    / "stage26_3b_cpu_memory_profile"
    / "stage26_3_memory_summary.csv"
)

PACKAGE_SIZES = (
    RESULT_ROOT
    / "stage26_3b_cpu_memory_profile"
    / "stage26_3_package_sizes.csv"
)

CAPACITY_BATCH = (
    RESULT_ROOT
    / "stage26_7c_cpu_capacity_scaling"
    / "stage26_7c_inference_batch_scaling.csv"
)

CAPACITY_TWO_CORE = (
    RESULT_ROOT
    / "stage26_7c_cpu_capacity_scaling"
    / "stage26_7c_two_core_scaling.csv"
)

GROUP_B_CAPACITY = (
    RESULT_ROOT
    / "stage26_7c_cpu_capacity_scaling"
    / "stage26_7c_group_b_component_capacity.csv"
)

PARETO = (
    RESULT_ROOT
    / "stage26_6d_cpu_pareto_point_estimate"
    / "stage26_6d_point_estimate_frontiers.csv"
)

REP_SENS_BATCH = (
    RESULT_ROOT
    / "stage26_8c1_paired_representation_sensitivity"
    / "evidence"
    / "stage26_8c1_batch_sensitivity_summary.csv"
)

REP_SENS_PAIRS = (
    RESULT_ROOT
    / "stage26_8c1_paired_representation_sensitivity"
    / "evidence"
    / "stage26_8c1_pair_results.csv"
)

REP_SENS_AUDIT = (
    RESULT_ROOT
    / "stage26_8c1_paired_representation_sensitivity"
    / "stage26_8c1_sensitivity_audit.json"
)

E2E_DECISION = (
    RESULT_ROOT
    / "stage26_5a_e2e_availability_closure"
    / "stage26_5a_e2e_availability_decision.json"
)

RAW_EXTRACTION_SUMMARY = (
    RESULT_ROOT
    / "stage26_4b3_cpu1_extraction_timing"
    / "stage26_4b3_cpu1_extraction_summary.json"
)

REPRESENTATION_HISTORICAL_SUMMARY = (
    RESULT_ROOT
    / "stage26_4c3_cpu1_representation_timing"
    / "stage26_4c3_cpu1_representation_summary.csv"
)

# -----------------------------------------------------------------------------
# Historical source anchors
# -----------------------------------------------------------------------------

SOURCE_SPECS = [
    (
        "warm_cpu_point_estimates",
        WARM_STATUS,
        "7ffdba3f4ca4ea5cc53097d62aaf27957009b9f6",
    ),
    (
        "corrected_warm_cpu_uncertainty",
        WARM_CORRECTED_UNCERTAINTY,
        "a484148cd8ea7d7c89604d3a015f73bd6f1bc81a",
    ),
    (
        "cold_start_implementation",
        COLD_IMPLEMENTATION,
        "46379b6d036008db4d60b056a66f4c01383e3298",
    ),
    (
        "cold_start_point_estimates",
        COLD_SUMMARY,
        "46379b6d036008db4d60b056a66f4c01383e3298",
    ),
    (
        "cpu_memory_summary",
        MEMORY_SUMMARY,
        "347d93f21d454cc5bda2c45889c890c67cdf0ecc",
    ),
    (
        "deployment_package_sizes",
        PACKAGE_SIZES,
        "347d93f21d454cc5bda2c45889c890c67cdf0ecc",
    ),
    (
        "capacity_batch_scaling",
        CAPACITY_BATCH,
        "9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3",
    ),
    (
        "capacity_two_core_scaling",
        CAPACITY_TWO_CORE,
        "9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3",
    ),
    (
        "group_b_component_capacity",
        GROUP_B_CAPACITY,
        "9b6c9d16762ebf314ac3f0a160b0bb4f77e15ca3",
    ),
    (
        "pareto_point_estimate",
        PARETO,
        "ff9d329785c6cd30d273729356f402e59dc4e844",
    ),
    (
        "representation_sensitivity_batch_summary",
        REP_SENS_BATCH,
        "fd4497e333ebeb72b1eb188ed64ca2cdd0f08567",
    ),
    (
        "representation_sensitivity_pair_results",
        REP_SENS_PAIRS,
        "fd4497e333ebeb72b1eb188ed64ca2cdd0f08567",
    ),
    (
        "representation_sensitivity_audit",
        REP_SENS_AUDIT,
        "fd4497e333ebeb72b1eb188ed64ca2cdd0f08567",
    ),
    (
        "complete_e2e_closure",
        E2E_DECISION,
        "92495eac2c9202973ff4d889e3c12669c06e9ef1",
    ),
    (
        "raw_extraction_component",
        RAW_EXTRACTION_SUMMARY,
        "55aba2e1cc08385659479c6275234dc23b11d231",
    ),
    (
        "historical_representation_component",
        REPRESENTATION_HISTORICAL_SUMMARY,
        "aee59fabab2465cca7c80904279bb6d3ef23894f",
    ),
]

# -----------------------------------------------------------------------------
# Output
# -----------------------------------------------------------------------------

OUT_DIR = RESULT_ROOT / "stage26_8d3_cpu_publication_tables"

TABLE_PATHS = {
    "T26_CPU_WARM_INFERENCE": OUT_DIR / "T26_CPU_WARM_INFERENCE.csv",
    "T26_CPU_MEMORY_PACKAGE": OUT_DIR / "T26_CPU_MEMORY_PACKAGE.csv",
    "T26_CPU_COLD_START": OUT_DIR / "T26_CPU_COLD_START.csv",
    "T26_CPU_CAPACITY_SCALING": OUT_DIR / "T26_CPU_CAPACITY_SCALING.csv",
    "T26_COMPONENT_MEASUREMENTS": OUT_DIR / "T26_COMPONENT_MEASUREMENTS.csv",
    "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO": (
        OUT_DIR / "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO.csv"
    ),
    "T26_PARETO": OUT_DIR / "T26_PARETO.csv",
    "T26_REPRESENTATION_SENSITIVITY": (
        OUT_DIR / "T26_REPRESENTATION_SENSITIVITY.csv"
    ),
}

INDEX_PATH = OUT_DIR / "stage26_8d3_cpu_publication_tables_index.json"
RECEIPT_PATH = OUT_DIR / "stage26_8d3_cpu_publication_tables_receipt.json"
MANIFEST_PATH = OUT_DIR / "stage26_8d3_cpu_publication_tables_manifest.json"

COMMIT_MESSAGE = "stage26: generate CPU publication tables"


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):
    print("\n" + "=" * 124)
    print(text)
    print("=" * 124)


def run(cmd, *, cwd=REPO, check=True, env=None):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
        env=env,
    )
    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}\n{p.stdout}"
        )
    return p.stdout.strip()


def git(*args, check=True, env=None):
    return run(["git", *args], check=check, env=env)


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            block = f.read(8 * 1024 * 1024)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def read_csv(path):
    with Path(path).open("r", encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, sort_keys=True, allow_nan=False)
        f.write("\n")
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def atomic_csv(path, columns, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with tmp.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=columns,
            extrasaction="raise",
            lineterminator="\n",
        )
        writer.writeheader()
        for row in rows:
            missing = [column for column in columns if column not in row]
            if missing:
                raise RuntimeError(
                    f"Row for {path.name} is missing columns: {missing}"
                )
            writer.writerow({column: row[column] for column in columns})
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def git_blob_bytes(ref, rel_path):
    p = subprocess.run(
        ["git", "show", f"{ref}:{rel_path}"],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )
    if p.returncode != 0:
        raise RuntimeError(
            f"Could not read {rel_path} from {ref}:\n"
            + p.stderr.decode("utf-8", errors="replace")
        )
    return p.stdout


def verify_historical_identity(label, path, anchor):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    rel = str(path.relative_to(REPO))
    current = path.read_bytes()
    historical = git_blob_bytes(anchor, rel)
    current_sha = sha256_bytes(current)
    historical_sha = sha256_bytes(historical)
    same = current == historical
    print(f"{'PASS' if same else 'FAIL'} {label}")
    print("  path      :", rel)
    print("  anchor    :", anchor)
    print("  historical:", historical_sha)
    print("  current   :", current_sha)
    if not same:
        raise RuntimeError(
            f"Source changed since durable scientific anchor: {rel}"
        )
    return {
        "label": label,
        "path": rel,
        "anchor": anchor,
        "sha256": current_sha,
        "byte_identical_to_anchor": True,
    }


def table_spec(schema, table_id):
    matches = [item for item in schema["tables"] if item["id"] == table_id]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one effective table spec for {table_id}; found {len(matches)}."
        )
    return matches[0]


def index_unique(rows, key, label):
    out = {}
    for row in rows:
        value = row[key]
        if value in out:
            raise RuntimeError(f"Duplicate {label} key: {value}")
        out[value] = row
    return out


def parse_int(value, label):
    try:
        return int(value)
    except Exception as exc:
        raise RuntimeError(f"Invalid integer for {label}: {value!r}") from exc


def parse_bool(value, label):
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text == "true":
        return True
    if text == "false":
        return False
    raise RuntimeError(f"Invalid boolean for {label}: {value!r}")


def decimal_value(value, label):
    text = str(value).strip()
    if text == "":
        return None
    try:
        return Decimal(text)
    except InvalidOperation as exc:
        raise RuntimeError(f"Invalid decimal for {label}: {value!r}") from exc


def numeric_equal(a, b, label):
    da = decimal_value(a, label + " left")
    db = decimal_value(b, label + " right")
    if da is None or db is None:
        return da is None and db is None
    return da == db


def bytes_to_mib_text(value, label):
    text = str(value).strip()
    if text == "":
        return ""
    d = decimal_value(text, label)
    out = d / Decimal(1048576)
    # Deterministic decimal rendering; conversion only, no aggregation.
    rendered = format(out, ".12f").rstrip("0").rstrip(".")
    return rendered if rendered else "0"


def target_sort_key(target_id):
    if target_id not in TARGET_RANK:
        raise RuntimeError(f"Unexpected target_id: {target_id}")
    return TARGET_RANK[target_id]


def batch_sort_key(batch_size):
    batch = int(batch_size)
    if batch not in BATCH_RANK:
        raise RuntimeError(f"Unexpected batch size: {batch}")
    return BATCH_RANK[batch]


def memory_status_from_counts(row):
    planned = parse_int(row["planned_repetitions"], "planned_repetitions")
    passed = parse_int(row["pass_repetitions"], "pass_repetitions")
    oom = parse_int(
        row["resource_limit_oom_repetitions"],
        "resource_limit_oom_repetitions",
    )
    timeout = parse_int(
        row["timeout_resource_limit_repetitions"],
        "timeout_resource_limit_repetitions",
    )

    if passed + oom + timeout != planned:
        raise RuntimeError(
            "Stage26-3B repetition counts do not exhaust the frozen condition: "
            f"{row.get('condition_id')}"
        )

    if passed == planned and oom == 0 and timeout == 0:
        return "PASS", "NONE"
    if oom == planned and passed == 0 and timeout == 0:
        return "RESOURCE_LIMIT_OOM", "RESOURCE_LIMIT_OOM"
    if timeout == planned and passed == 0 and oom == 0:
        return "TIMEOUT_RESOURCE_LIMIT", "TIMEOUT_RESOURCE_LIMIT"

    outcomes = []
    if passed:
        outcomes.append(f"PASS:{passed}")
    if oom:
        outcomes.append(f"RESOURCE_LIMIT_OOM:{oom}")
    if timeout:
        outcomes.append(f"TIMEOUT_RESOURCE_LIMIT:{timeout}")
    return "MIXED_FROZEN_OUTCOME", ";".join(outcomes)


# =============================================================================
# 2. DURABLE STATE GATE
# =============================================================================

banner("STAGE26-8D3 :: DURABLE STATE GATE")

git("fetch", "origin", "main")

head = git("rev-parse", "HEAD")
remote = git("rev-parse", "origin/main")
status = git("status", "--porcelain")

print("Expected parent:", EXPECTED_PARENT)
print("Local HEAD     :", head)
print("origin/main    :", remote)
print("Repo clean     :", status == "")

if head != EXPECTED_PARENT:
    raise RuntimeError("Unexpected local HEAD before Stage26-8D3.")
if remote != EXPECTED_PARENT:
    raise RuntimeError("origin/main changed before Stage26-8D3.")
if status:
    raise RuntimeError("Repository must be clean before Stage26-8D3.")
if OUT_DIR.exists():
    raise RuntimeError(
        f"Stage26-8D3 output directory already exists: {OUT_DIR}\n"
        "Do not overwrite durable publication tables."
    )


# =============================================================================
# 3. EFFECTIVE SCHEMA GATE
# =============================================================================

banner("STAGE26-8D3 :: EFFECTIVE SCHEMA GATE")

if not EFFECTIVE_SCHEMA.is_file():
    raise FileNotFoundError(EFFECTIVE_SCHEMA)

schema_sha = sha256_file(EFFECTIVE_SCHEMA)
print("Expected effective schema SHA256:", EXPECTED_EFFECTIVE_SCHEMA_SHA256)
print("Actual effective schema SHA256  :", schema_sha)

if schema_sha != EXPECTED_EFFECTIVE_SCHEMA_SHA256:
    raise RuntimeError("Effective Stage26-8D2A schema SHA mismatch.")

schema = read_json(EFFECTIVE_SCHEMA)

expected_table_ids = list(TABLE_PATHS)
actual_table_ids = [item["id"] for item in schema["tables"]]

print("Table IDs:")
for table_id in actual_table_ids:
    print(" ", table_id)

if actual_table_ids != expected_table_ids:
    raise RuntimeError(
        "Effective schema table identity/order differs from the frozen eight-table universe."
    )

if schema.get("publication_label") != PUBLICATION_LABEL:
    raise RuntimeError("Unexpected effective publication label.")

if schema["scope"].get("gpu_allowed") is not False:
    raise RuntimeError("GPU must remain OFF for Stage26-8D3.")


# =============================================================================
# 4. SOURCE BYTE-IDENTITY GATES
# =============================================================================

banner("STAGE26-8D3 :: SOURCE BYTE-IDENTITY GATES")

source_identity = []
for label, path, anchor in SOURCE_SPECS:
    source_identity.append(
        verify_historical_identity(label, path, anchor)
    )


# =============================================================================
# 5. LOAD SMALL COMMITTED SUMMARY ARTIFACTS ONLY
# =============================================================================

banner("STAGE26-8D3 :: LOAD COMMITTED SUMMARY ARTIFACTS")

warm_status_rows = read_csv(WARM_STATUS)
warm_uncertainty_rows = read_csv(WARM_CORRECTED_UNCERTAINTY)
cold_impl = read_json(COLD_IMPLEMENTATION)
cold_rows = read_csv(COLD_SUMMARY)
memory_rows = read_csv(MEMORY_SUMMARY)
package_rows = read_csv(PACKAGE_SIZES)
capacity_batch_rows = read_csv(CAPACITY_BATCH)
capacity_two_core_rows = read_csv(CAPACITY_TWO_CORE)
group_b_rows = read_csv(GROUP_B_CAPACITY)
pareto_rows = read_csv(PARETO)
rep_batch_rows = read_csv(REP_SENS_BATCH)
rep_pair_rows = read_csv(REP_SENS_PAIRS)
rep_audit = read_json(REP_SENS_AUDIT)

print("Warm status rows              :", len(warm_status_rows))
print("Warm uncertainty rows         :", len(warm_uncertainty_rows))
print("Cold-start summary rows       :", len(cold_rows))
print("Memory condition rows         :", len(memory_rows))
print("Package-size rows             :", len(package_rows))
print("Capacity batch rows           :", len(capacity_batch_rows))
print("Capacity two-core rows        :", len(capacity_two_core_rows))
print("Group-B component-ratio rows  :", len(group_b_rows))
print("Pareto rows                   :", len(pareto_rows))
print("Sensitivity batch rows        :", len(rep_batch_rows))
print("Sensitivity pair rows         :", len(rep_pair_rows))

if len(warm_status_rows) != 80:
    raise RuntimeError("Expected exactly 80 Stage26-2 CPU conditions.")
if len(warm_uncertainty_rows) != 80:
    raise RuntimeError("Expected exactly 80 Stage26-6F1 CPU conditions.")
if len(memory_rows) != 80:
    raise RuntimeError("Expected exactly 80 Stage26-3B memory conditions.")
if len(package_rows) != 8:
    raise RuntimeError("Expected exactly 8 Stage26-3B package-size rows.")
if len(capacity_batch_rows) != 80:
    raise RuntimeError("Expected exactly 80 Stage26-7C batch-scaling rows.")
if len(rep_batch_rows) != 5:
    raise RuntimeError("Expected exactly five frozen Stage26-8 sensitivity batches.")
if len(rep_pair_rows) != 25:
    raise RuntimeError("Expected exactly 25 frozen Stage26-8 paired blocks.")


# =============================================================================
# 6. COMMON TARGET/GROUP MAP
# =============================================================================

banner("STAGE26-8D3 :: COMMON TARGET/GROUP MAP")

groups_seen = defaultdict(set)
for row in warm_status_rows:
    groups_seen[row["target_id"]].add(row["comparison_group"])

if set(groups_seen) != set(TARGET_ORDER):
    raise RuntimeError(
        "Warm-inference target universe differs from the frozen eight-target universe."
    )

group_by_target = {}
for target in TARGET_ORDER:
    values = groups_seen[target]
    if len(values) != 1:
        raise RuntimeError(f"Target has non-unique comparison group: {target} -> {values}")
    group_by_target[target] = next(iter(values))
    print(f"{target:44s} -> {group_by_target[target]}")


# =============================================================================
# 7. T1 — CPU ISOLATED WARM INFERENCE
# =============================================================================

banner("STAGE26-8D3 :: BUILD T26_CPU_WARM_INFERENCE")

warm_unc_by_condition = index_unique(
    warm_uncertainty_rows,
    "condition_id",
    "warm corrected uncertainty",
)

warm_stage2_condition_ids = {row["condition_id"] for row in warm_status_rows}
if set(warm_unc_by_condition) != warm_stage2_condition_ids:
    missing = sorted(warm_stage2_condition_ids - set(warm_unc_by_condition))
    extra = sorted(set(warm_unc_by_condition) - warm_stage2_condition_ids)
    raise RuntimeError(
        "Stage26-2 / Stage26-6F1 condition-ID universe mismatch. "
        f"missing={missing} extra={extra}"
    )

T1 = []

for src in warm_status_rows:
    condition_id = src["condition_id"]
    status_value = src["status"]
    unc = warm_unc_by_condition.get(condition_id)

    if unc is None:
        raise RuntimeError(
            f"Warm condition lacks Stage26-6F1 companion row: {condition_id}"
        )

    if status_value == "PASS":
        # Stage26-6F1 carries full identity + corrected uncertainty for PASS rows.
        identity_fields = [
            ("target_id", "target_id"),
            ("comparison_group", "comparison_group"),
            ("hardware_mode", "hardware_mode"),
            ("thread_count", "thread_count"),
            ("batch_size", "batch_size"),
            ("status", "status"),
        ]
        for left, right in identity_fields:
            if src[left] != unc[right]:
                raise RuntimeError(
                    f"Warm Stage26-2/6F1 PASS identity mismatch for {condition_id}: "
                    f"{left}={src[left]!r} vs {unc[right]!r}"
                )

        if src["timed_runs_observed"] != unc["n"]:
            raise RuntimeError(
                f"Warm PASS observation-count mismatch for {condition_id}."
            )

        point_pairs = [
            ("p50_batch_latency_ms", "p50_point_ms"),
            ("p95_batch_latency_ms", "p95_point_ms"),
            ("p99_batch_latency_ms", "p99_point_ms_if_n_gte_100"),
            ("median_throughput_flows_per_second", "median_throughput_point"),
        ]
        for a, b in point_pairs:
            if not numeric_equal(src[a], unc[b], f"{condition_id}:{a}"):
                raise RuntimeError(
                    f"Warm point estimate changed between Stage26-2 and 6F1: "
                    f"{condition_id} {a}"
                )

        if unc["corrected_ci_status"] != (
            "PROTOCOL_CERTIFIED_DERIVED_UNCERTAINTY"
        ):
            raise RuntimeError(
                f"Unexpected Stage26-6F1 corrected CI status for {condition_id}: "
                f"{unc['corrected_ci_status']}"
            )

    else:
        # Stage26-6F1 intentionally stores only condition_id + resource-limit
        # status for non-PASS rows. Full identity remains authoritative in
        # Stage26-2 and uncertainty is explicitly NOT_APPLICABLE_NON_PASS.
        if unc["status"] != status_value:
            raise RuntimeError(
                f"Warm Stage26-2/6F1 non-PASS status mismatch for {condition_id}: "
                f"{status_value!r} vs {unc['status']!r}"
            )

        for field in [
            "target_id",
            "comparison_group",
            "hardware_mode",
            "thread_count",
            "batch_size",
            "n",
        ]:
            if str(unc.get(field, "")).strip() != "":
                raise RuntimeError(
                    f"Stage26-6F1 non-PASS field must remain blank for "
                    f"{condition_id}: {field}={unc.get(field)!r}"
                )

        if unc.get("historical_ci_status") != "NOT_APPLICABLE_NON_PASS":
            raise RuntimeError(
                f"Unexpected historical CI status for non-PASS {condition_id}: "
                f"{unc.get('historical_ci_status')!r}"
            )

        if unc.get("corrected_ci_status") != "NOT_APPLICABLE_NON_PASS":
            raise RuntimeError(
                f"Unexpected corrected CI status for non-PASS {condition_id}: "
                f"{unc.get('corrected_ci_status')!r}"
            )

        if src["timed_runs_observed"] != "0":
            raise RuntimeError(
                f"Non-PASS Stage26-2 condition must have zero timed observations: "
                f"{condition_id} -> {src['timed_runs_observed']!r}"
            )

        for field in [
            "p50_batch_latency_ms",
            "p95_batch_latency_ms",
            "p99_batch_latency_ms",
            "median_throughput_flows_per_second",
        ]:
            if str(src.get(field, "")).strip() != "":
                raise RuntimeError(
                    f"Non-PASS Stage26-2 metric must remain unavailable for "
                    f"{condition_id}: {field}={src.get(field)!r}"
                )

    def u(field):
        # Publication uncertainty is permitted only for Stage26-6F1 PASS rows.
        return unc.get(field, "") if status_value == "PASS" else ""

    T1.append(
        {
            "target_id": src["target_id"],
            "group": src["comparison_group"],
            "cpu_mode": src["hardware_mode"],
            "batch_size": src["batch_size"],
            "status": status_value,
            "n_timed_observations": src["timed_runs_observed"],
            "p50_latency_ms": src["p50_batch_latency_ms"],
            "p50_ci95_low_ms": u("corrected_p50_ci_low_ms"),
            "p50_ci95_high_ms": u("corrected_p50_ci_high_ms"),
            "p95_latency_ms": src["p95_batch_latency_ms"],
            "p95_ci95_low_ms": u("corrected_p95_ci_low_ms"),
            "p95_ci95_high_ms": u("corrected_p95_ci_high_ms"),
            "p99_latency_ms": src["p99_batch_latency_ms"],
            "p99_ci95_low_ms": u("corrected_p99_ci_low_ms_if_n_gte_100"),
            "p99_ci95_high_ms": u("corrected_p99_ci_high_ms_if_n_gte_100"),
            "median_throughput_samples_per_s": (
                src["median_throughput_flows_per_second"]
            ),
            "throughput_ci95_low_samples_per_s": (
                u("corrected_median_throughput_ci_low")
            ),
            "throughput_ci95_high_samples_per_s": (
                u("corrected_median_throughput_ci_high")
            ),
            "resource_limit_outcome": (
                "NONE" if status_value == "PASS" else status_value
            ),
        }
    )

T1.sort(
    key=lambda row: (
        target_sort_key(row["target_id"]),
        CPU_ORDER[row["cpu_mode"]],
        batch_sort_key(row["batch_size"]),
    )
)

if len(T1) != 80:
    raise RuntimeError("T26_CPU_WARM_INFERENCE must contain 80 rows.")


# =============================================================================
# 8. T2 — CPU MEMORY + DEPLOYMENT PACKAGE
# =============================================================================

banner("STAGE26-8D3 :: BUILD T26_CPU_MEMORY_PACKAGE")

package_by_target = index_unique(package_rows, "target_id", "package size")

if set(package_by_target) != set(TARGET_ORDER):
    raise RuntimeError("Package-size target universe mismatch.")

memory_spec = table_spec(schema, "T26_CPU_MEMORY_PACKAGE")
metric_specs = memory_spec["frozen_primary_memory_metrics"]
metric_rank = {
    item["output_name"]: i for i, item in enumerate(metric_specs)
}

T2 = []

for src in memory_rows:
    target = src["target_id"]
    if target not in package_by_target:
        raise RuntimeError(f"Memory target lacks package-size row: {target}")
    if src["comparison_group"] != group_by_target[target]:
        raise RuntimeError(f"Memory comparison-group mismatch for {target}.")

    memory_status, resource_outcome = memory_status_from_counts(src)

    for metric in metric_specs:
        source_field = metric["source_field"]
        output_name = metric["output_name"]
        if source_field not in src:
            raise RuntimeError(
                f"Frozen memory source field missing: {source_field}"
            )

        T2.append(
            {
                "target_id": target,
                "group": src["comparison_group"],
                "cpu_mode": src["hardware_mode"],
                "batch_size": src["batch_size"],
                "deployment_package_size_mib": (
                    package_by_target[target]["deployment_package_size_mib"]
                ),
                "memory_metric_name": output_name,
                "memory_value_mib": bytes_to_mib_text(
                    src[source_field],
                    f"{src['condition_id']}:{source_field}",
                ),
                "planned_repetitions": src["planned_repetitions"],
                "pass_repetitions": src["pass_repetitions"],
                "resource_limit_oom_repetitions": (
                    src["resource_limit_oom_repetitions"]
                ),
                "timeout_resource_limit_repetitions": (
                    src["timeout_resource_limit_repetitions"]
                ),
                "memory_status": memory_status,
                "resource_limit_outcome": resource_outcome,
            }
        )

T2.sort(
    key=lambda row: (
        target_sort_key(row["target_id"]),
        CPU_ORDER[row["cpu_mode"]],
        batch_sort_key(row["batch_size"]),
        metric_rank[row["memory_metric_name"]],
    )
)

expected_t2_rows = 80 * len(metric_specs)
if len(T2) != expected_t2_rows:
    raise RuntimeError(
        f"T26_CPU_MEMORY_PACKAGE expected {expected_t2_rows} rows; got {len(T2)}."
    )


# =============================================================================
# 9. T3 — CPU COLD START, POINT ESTIMATES ONLY
# =============================================================================

banner("STAGE26-8D3 :: BUILD T26_CPU_COLD_START")

if cold_impl.get("gpu") is not False:
    raise RuntimeError("Cold-start implementation unexpectedly indicates GPU use.")

cold_cpu_mode = cold_impl["hardware_mode"]
expected_cold_n = int(cold_impl["repetitions_per_target"])

if cold_cpu_mode != "CPU_1_PHYSICAL_CORE":
    raise RuntimeError("Unexpected frozen cold-start hardware mode.")
if expected_cold_n != 20:
    raise RuntimeError("Unexpected frozen cold-start repetition count.")

cold_targets = {row["target_id"] for row in cold_rows}
if cold_targets != set(TARGET_ORDER):
    raise RuntimeError("Cold-start target universe mismatch.")

T3 = []

for src in cold_rows:
    n = parse_int(src["n"], "cold-start n")
    if n != expected_cold_n:
        raise RuntimeError(
            f"Cold-start summary is incomplete for {src['target_id']} / {src['component']}: "
            f"n={n}, expected={expected_cold_n}"
        )

    component = src["component"]
    if component not in COLD_COMPONENT_RANK:
        raise RuntimeError(f"Unexpected cold-start component: {component}")

    T3.append(
        {
            "target_id": src["target_id"],
            "group": group_by_target[src["target_id"]],
            "cpu_mode": cold_cpu_mode,
            "component": component,
            "n_observations": src["n"],
            "p50_ms": src["p50_ms"],
            "p95_ms": src["p95_ms"],
            "status": "PASS",
        }
    )

T3.sort(
    key=lambda row: (
        target_sort_key(row["target_id"]),
        COLD_COMPONENT_RANK[row["component"]],
    )
)

expected_t3_rows = len(TARGET_ORDER) * len(COLD_COMPONENT_ORDER)
if len(T3) != expected_t3_rows:
    raise RuntimeError(
        f"T26_CPU_COLD_START expected {expected_t3_rows} rows; got {len(T3)}."
    )


# =============================================================================
# 10. T4 — CPU CAPACITY / SCALING, IMMUTABLE STAGE26-7C VALUES
# =============================================================================

banner("STAGE26-8D3 :: BUILD T26_CPU_CAPACITY_SCALING")

batch_keyed = {}
for src in capacity_batch_rows:
    target = src["target_id"]
    batch = parse_int(src["batch_size"], "capacity batch")
    mode = src["hardware_mode"]
    if target not in TARGET_RANK:
        raise RuntimeError(f"Unexpected Stage26-7C target: {target}")
    if mode not in CPU_ORDER:
        raise RuntimeError(f"Unexpected Stage26-7C hardware mode: {mode}")
    key = (target, batch, mode)
    if key in batch_keyed:
        raise RuntimeError(f"Duplicate Stage26-7C batch row: {key}")
    batch_keyed[key] = src

expected_batch_keys = {
    (target, batch, mode)
    for target in TARGET_ORDER
    for batch in BATCH_ORDER
    for mode in CPU_ORDER
}

if set(batch_keyed) != expected_batch_keys:
    missing = sorted(expected_batch_keys - set(batch_keyed))
    extra = sorted(set(batch_keyed) - expected_batch_keys)
    raise RuntimeError(
        f"Stage26-7C batch-scaling key universe mismatch. missing={missing[:5]} extra={extra[:5]}"
    )

two_core_by_key = {}
for src in capacity_two_core_rows:
    key = (src["target_id"], parse_int(src["batch_size"], "two-core batch"))
    if key in two_core_by_key:
        raise RuntimeError(f"Duplicate Stage26-7C two-core row: {key}")
    if parse_bool(src["uncertainty_propagated"], "two-core uncertainty"):
        raise RuntimeError("Stage26-7C two-core row unexpectedly propagates uncertainty.")
    two_core_by_key[key] = src

T4 = []

for target in TARGET_ORDER:
    for batch in BATCH_ORDER:
        cpu1 = batch_keyed[(target, batch, "CPU_1_PHYSICAL_CORE")]
        cpu2 = batch_keyed[(target, batch, "CPU_2_PHYSICAL_CORE")]
        two = two_core_by_key.get((target, batch))

        if parse_bool(cpu1["uncertainty_propagated"], "CPU1 batch uncertainty"):
            raise RuntimeError("Stage26-7C CPU1 batch row unexpectedly propagates uncertainty.")
        if parse_bool(cpu2["uncertainty_propagated"], "CPU2 batch uncertainty"):
            raise RuntimeError("Stage26-7C CPU2 batch row unexpectedly propagates uncertainty.")

        both_pass = cpu1["status"] == "PASS" and cpu2["status"] == "PASS"
        if both_pass and two is None:
            raise RuntimeError(
                f"Matched PASS CPU1/CPU2 condition lacks immutable Stage26-7C two-core row: "
                f"{target}, batch={batch}"
            )

        T4.append(
            {
                "target_id": target,
                "group": group_by_target[target],
                "batch_size": str(batch),
                "cpu1_status": cpu1["status"],
                "cpu2_status": cpu2["status"],
                "cpu1_throughput_multiplier_vs_b1": (
                    cpu1["throughput_multiplier_vs_B1"]
                ),
                "cpu2_throughput_multiplier_vs_b1": (
                    cpu2["throughput_multiplier_vs_B1"]
                ),
                "cpu2_over_cpu1_speedup": (
                    "" if two is None else two["two_core_speedup"]
                ),
                "physical_core_parallel_efficiency": (
                    "" if two is None else two["two_core_parallel_efficiency"]
                ),
            }
        )

if len(T4) != 40:
    raise RuntimeError("T26_CPU_CAPACITY_SCALING must contain 40 rows.")


# =============================================================================
# 11. T5 — COMPONENT BOUNDARY / E2E AVAILABILITY MATRIX
# =============================================================================

banner("STAGE26-8D3 :: BUILD T26_COMPONENT_MEASUREMENTS")

component_spec = table_spec(schema, "T26_COMPONENT_MEASUREMENTS")
fixed_component_rows = component_spec.get("fixed_rows")

if not isinstance(fixed_component_rows, list) or len(fixed_component_rows) != 4:
    raise RuntimeError("Effective T26_COMPONENT_MEASUREMENTS fixed-row lock is missing.")

T5 = []
for src in fixed_component_rows:
    row = dict(src)
    if row.get("complete_e2e_available") is not False:
        raise RuntimeError(
            "Component-boundary table unexpectedly allows complete E2E measurement."
        )
    T5.append(row)


# =============================================================================
# 12. T6 — GROUP-B INFERENCE / REPRESENTATION COMPONENT CAPACITY RATIO
# =============================================================================

banner("STAGE26-8D3 :: BUILD T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO")

T6 = []

for src in group_b_rows:
    if src["hardware_mode"] != "CPU_1_PHYSICAL_CORE":
        raise RuntimeError("Group-B component ratio contains unexpected hardware mode.")
    if parse_bool(src["uncertainty_propagated"], "Group-B ratio uncertainty"):
        raise RuntimeError("Group-B component ratio unexpectedly propagates uncertainty.")
    if src["target_id"] not in {
        "STAGE20_MASKED_CNN_V1",
        "STAGE21_MASKED_VIT_V1",
    }:
        raise RuntimeError(f"Unexpected Group-B ratio target: {src['target_id']}")

    T6.append(
        {
            "target_id": src["target_id"],
            "batch_size": src["batch_size"],
            "inference_over_representation_capacity_ratio": (
                src["component_capacity_ratio_inference_over_representation"]
            ),
            "publication_label": PUBLICATION_LABEL,
        }
    )

T6.sort(
    key=lambda row: (
        target_sort_key(row["target_id"]),
        batch_sort_key(row["batch_size"]),
    )
)

if len(T6) != 7:
    raise RuntimeError("Expected exactly 7 frozen Group-B matched component-ratio rows.")


# =============================================================================
# 13. T7 — IMMUTABLE WITHIN-GROUP PARETO POINT ESTIMATE
# =============================================================================

banner("STAGE26-8D3 :: BUILD T26_PARETO")

T7 = []

for src in pareto_rows:
    target = src["target_id"]
    if target not in {
        "STAGE16_XGBOOST_TUNED",
        "STAGE16_LIGHTGBM_TUNED",
        "STAGE16_CATBOOST_TUNED",
        "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
        "STAGE20_MASKED_CNN_V1",
        "STAGE21_MASKED_VIT_V1",
    }:
        raise RuntimeError(f"Unexpected Pareto target: {target}")

    member = parse_bool(src["frontier_member"], "Pareto frontier_member")

    T7.append(
        {
            "group": src["comparison_group"],
            "target_id": target,
            "pr_auc": src["pr_auc"],
            "cpu1_batch1_p95_latency_ms": (
                src["p95_cpu1_batch1_inference_latency_ms"]
            ),
            "pareto_status": (
                "FRONTIER_MEMBER" if member else "NOT_FRONTIER_MEMBER"
            ),
        }
    )

T7.sort(key=lambda row: target_sort_key(row["target_id"]))

if len(T7) != 6:
    raise RuntimeError("T26_PARETO must contain exactly six primary-architecture rows.")


# =============================================================================
# 14. T8 — PAIRED REPRESENTATION IMPLEMENTATION SENSITIVITY
# =============================================================================

banner("STAGE26-8D3 :: BUILD T26_REPRESENTATION_SENSITIVITY")

if rep_audit.get("status") != (
    "PASS_PAIRED_4C3_IMPLEMENTATION_SENSITIVITY_AUDIT_COMPLETED"
):
    raise RuntimeError("Unexpected Stage26-8 representation-sensitivity audit status.")

if rep_audit["statistical_policy"].get("bootstrap") is not False:
    raise RuntimeError("Stage26-8 sensitivity unexpectedly permits bootstrap.")
if rep_audit["statistical_policy"].get("hypothesis_test") is not False:
    raise RuntimeError("Stage26-8 sensitivity unexpectedly permits hypothesis testing.")
if rep_audit["statistical_policy"].get("materiality_threshold") is not None:
    raise RuntimeError("Stage26-8 sensitivity unexpectedly has a materiality threshold.")
if rep_audit["execution_geometry"].get("all_pair_fingerprints_equal") is not True:
    raise RuntimeError("Stage26-8 audit does not certify equal pair fingerprints.")

supported_interpretation = (
    rep_audit["descriptive_interpretation"]["supported_statement"]
)
audit_batches = rep_audit["descriptive_batch_results"]

pairs_by_batch = defaultdict(list)
for pair in rep_pair_rows:
    batch = parse_int(pair["batch_size"], "sensitivity pair batch")
    pairs_by_batch[batch].append(pair)

T8 = []

for src in rep_batch_rows:
    batch = parse_int(src["batch_size"], "sensitivity summary batch")
    if batch not in BATCH_RANK:
        raise RuntimeError(f"Unexpected sensitivity batch: {batch}")

    if parse_bool(src["bootstrap"], "sensitivity bootstrap"):
        raise RuntimeError("Sensitivity batch row unexpectedly indicates bootstrap.")
    if parse_bool(src["hypothesis_test"], "sensitivity hypothesis_test"):
        raise RuntimeError("Sensitivity batch row unexpectedly indicates a hypothesis test.")
    if str(src["materiality_threshold"]).strip() != "":
        raise RuntimeError("Sensitivity batch row unexpectedly has a materiality threshold.")

    n_pairs = parse_int(src["complete_pair_count"], "complete_pair_count")
    batch_pairs = pairs_by_batch[batch]
    complete_pairs = [
        pair
        for pair in batch_pairs
        if parse_bool(pair["pair_complete"], "pair_complete")
    ]

    if len(complete_pairs) != n_pairs:
        raise RuntimeError(
            f"Sensitivity complete-pair count mismatch for batch {batch}."
        )

    fingerprint_equal_count = sum(
        1
        for pair in complete_pairs
        if parse_bool(pair["fingerprint_equal"], "fingerprint_equal")
    )

    if fingerprint_equal_count != n_pairs:
        raise RuntimeError(
            f"Sensitivity fingerprint equality failed for batch {batch}."
        )

    audit_row = audit_batches.get(str(batch))
    if audit_row is None:
        raise RuntimeError(f"Sensitivity audit lacks batch {batch}.")

    if not numeric_equal(
        src["throughput_ratio_median"],
        audit_row["throughput_ratio_V3_over_V2_median"],
        f"sensitivity throughput median batch {batch}",
    ):
        raise RuntimeError(
            f"Sensitivity batch-summary/audit throughput median mismatch: batch {batch}."
        )

    T8.append(
        {
            "batch_size": str(batch),
            "n_pairs": str(n_pairs),
            "fingerprints_equal_count": str(fingerprint_equal_count),
            "median_throughput_ratio_v3_over_v2": src["throughput_ratio_median"],
            "effect_direction": audit_row["throughput_direction"],
            "interpretation": supported_interpretation,
        }
    )

T8.sort(key=lambda row: batch_sort_key(row["batch_size"]))

if len(T8) != 5:
    raise RuntimeError("T26_REPRESENTATION_SENSITIVITY must contain five rows.")


# =============================================================================
# 15. WRITE TABLES USING EXACT EFFECTIVE-SCHEMA COLUMN ORDER
# =============================================================================

banner("STAGE26-8D3 :: WRITE EIGHT TABLES")

OUT_DIR.mkdir(parents=True, exist_ok=False)

TABLE_DATA = {
    "T26_CPU_WARM_INFERENCE": T1,
    "T26_CPU_MEMORY_PACKAGE": T2,
    "T26_CPU_COLD_START": T3,
    "T26_CPU_CAPACITY_SCALING": T4,
    "T26_COMPONENT_MEASUREMENTS": T5,
    "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO": T6,
    "T26_PARETO": T7,
    "T26_REPRESENTATION_SENSITIVITY": T8,
}

output_records = []

for table_id in expected_table_ids:
    spec = table_spec(schema, table_id)
    columns = spec["columns"]
    rows = TABLE_DATA[table_id]
    path = TABLE_PATHS[table_id]

    atomic_csv(path, columns, rows)

    # Re-open and verify exact header + row count.
    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        actual_header = next(reader)
        actual_row_count = sum(1 for _ in reader)

    if actual_header != columns:
        raise RuntimeError(f"Output header differs from effective schema: {table_id}")
    if actual_row_count != len(rows):
        raise RuntimeError(f"Output row count mismatch: {table_id}")

    record = {
        "table_id": table_id,
        "title": spec["title"],
        "repo_relative_path": str(path.relative_to(REPO)),
        "columns": columns,
        "row_count": len(rows),
        "sha256": sha256_file(path),
        "size_bytes": int(path.stat().st_size),
    }
    output_records.append(record)

    print(f"{table_id:50s} rows={len(rows):4d} SHA256={record['sha256']}")


# =============================================================================
# 16. PUBLICATION-SAFETY VALIDATION
# =============================================================================

banner("STAGE26-8D3 :: PUBLICATION-SAFETY VALIDATION")

# Cold-start table must never expose stored Stage26-1 CIs.
cold_columns = table_spec(schema, "T26_CPU_COLD_START")["columns"]
for forbidden in [
    "p50_bootstrap_ci95_low_ms",
    "p50_bootstrap_ci95_high_ms",
    "p95_bootstrap_ci95_low_ms",
    "p95_bootstrap_ci95_high_ms",
]:
    if forbidden in cold_columns:
        raise RuntimeError(f"Forbidden cold-start CI column survived: {forbidden}")

# Warm table must not contain historical Stage26-2 CI names.
warm_columns = table_spec(schema, "T26_CPU_WARM_INFERENCE")["columns"]
if any("historical" in column.lower() for column in warm_columns):
    raise RuntimeError("Historical Stage26-2 uncertainty leaked into warm publication table.")

# T4 and T6 values are copied, not recomputed.
if table_spec(schema, "T26_CPU_CAPACITY_SCALING")["rules"][0] != (
    "Copy immutable Stage26-7C descriptive values; do not recompute them."
):
    raise RuntimeError("Effective T4 immutable-copy rule changed.")

if table_spec(schema, "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO")[
    "frozen_formula"
] != (
    "inference_median_throughput_flows_per_second / "
    "representation_median_flows_per_second"
):
    raise RuntimeError("Effective T6 frozen formula changed.")

# T5 must explicitly retain complete E2E unavailability.
if any(row["complete_e2e_available"] is not False for row in T5):
    raise RuntimeError("T5 contains an invalid complete-E2E availability claim.")

# Pareto remains within-group only.
pareto_groups = {row["group"] for row in T7}
if pareto_groups != {"GROUP_A_DUPSAFE70", "GROUP_B_PACKET_IMAGE"}:
    raise RuntimeError("Unexpected Pareto group universe.")

print("Cold-start CIs absent                : PASS")
print("Historical Stage26-2 CIs absent      : PASS")
print("Stage26-7C ratios/scaling not redone : PASS")
print("Group-B ratio not inverted           : PASS")
print("Memory condition aggregation         : NONE")
print("Complete E2E                         : UNAVAILABLE")
print("Pareto recomputation                 : NONE")
print("Stage26-8 inference/bootstrap        : NONE")
print("GPU                                  : OFF")


# =============================================================================
# 17. WRITE INDEX / RECEIPT / MANIFEST
# =============================================================================

banner("STAGE26-8D3 :: WRITE INDEX / RECEIPT / MANIFEST")

index_payload = {
    "schema": "stage26_8d3_cpu_publication_tables_index_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "scientific_parent": EXPECTED_PARENT,
    "effective_schema_repo_relative_path": str(EFFECTIVE_SCHEMA.relative_to(REPO)),
    "effective_schema_sha256": EXPECTED_EFFECTIVE_SCHEMA_SHA256,
    "publication_label": PUBLICATION_LABEL,
    "table_count": len(output_records),
    "tables": output_records,
}
atomic_json(INDEX_PATH, index_payload)
index_sha = sha256_file(INDEX_PATH)

receipt_payload = {
    "schema": "stage26_8d3_cpu_publication_tables_receipt_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "scientific_parent": EXPECTED_PARENT,
    "effective_schema_sha256": EXPECTED_EFFECTIVE_SCHEMA_SHA256,
    "source_identity": source_identity,
    "table_count": len(output_records),
    "table_row_counts": {
        record["table_id"]: record["row_count"] for record in output_records
    },
    "deterministic_publication_transforms": [
        "SOURCE_COLUMN_SELECTION_AND_RENAMING",
        "EXACT_KEY_JOINS",
        "BYTES_TO_MIB_UNIT_CONVERSION",
        "MEMORY_STATUS_LABEL_FROM_FROZEN_REPETITION_COUNTS",
        "PAIR_FINGERPRINT_EQUALITY_COUNT_REQUIRED_BY_FROZEN_T8_SCHEMA",
    ],
    "scientific_safety": {
        "measured_values_changed": False,
        "timing_executed": False,
        "inference_executed": False,
        "model_loaded": False,
        "bootstrap_executed": False,
        "random_sampling_executed": False,
        "cold_start_ci_used": False,
        "historical_stage26_2_ci_used": False,
        "memory_aggregated_across_cpu_or_batch": False,
        "stage26_7c_recomputed": False,
        "group_b_ratio_recomputed": False,
        "group_b_ratio_inverted": False,
        "pareto_recomputed": False,
        "stage26_8_hypothesis_test": False,
        "stage26_8_materiality_threshold": False,
        "pcap_accessed": False,
        "labels_accessed": False,
        "gpu_used": False,
    },
    "complete_e2e": "UNAVAILABLE",
    "historical_stage26_4c3": "RETAINED_AS_ORIGINAL_MEASUREMENT",
    "stage26_8_representation_sensitivity": "DESCRIPTIVE_SEPARATE_NOT_REPLACEMENT",
}
atomic_json(RECEIPT_PATH, receipt_payload)
receipt_sha = sha256_file(RECEIPT_PATH)

manifest_files = []
for path in [*TABLE_PATHS.values(), INDEX_PATH, RECEIPT_PATH]:
    manifest_files.append(
        {
            "path": str(path.relative_to(REPO)),
            "sha256": sha256_file(path),
            "size_bytes": int(path.stat().st_size),
        }
    )

manifest_payload = {
    "schema": "stage26_8d3_cpu_publication_tables_manifest_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "scientific_parent": EXPECTED_PARENT,
    "files": manifest_files,
}
atomic_json(MANIFEST_PATH, manifest_payload)
manifest_sha = sha256_file(MANIFEST_PATH)

print("Index SHA256   :", index_sha)
print("Receipt SHA256 :", receipt_sha)
print("Manifest SHA256:", manifest_sha)


# =============================================================================
# 18. PRE-COMMIT GIT AUDIT
# =============================================================================

banner("STAGE26-8D3 :: PRE-COMMIT GIT AUDIT")

status_lines = [
    line for line in git("status", "--porcelain").splitlines() if line.strip()
]

for line in status_lines:
    print(line)

expected_prefix = "?? " + str(OUT_DIR.relative_to(REPO))

if not status_lines:
    raise RuntimeError("No Stage26-8D3 files detected by Git.")
if any(not line.startswith(expected_prefix) for line in status_lines):
    raise RuntimeError(
        "Unexpected repository modification detected before Stage26-8D3 commit."
    )


# =============================================================================
# 19. COMMIT
# =============================================================================

banner("STAGE26-8D3 :: COMMIT")

git("add", str(OUT_DIR.relative_to(REPO)))

staged = git("diff", "--cached", "--name-only").splitlines()
expected_staged = sorted(
    [
        *(str(path.relative_to(REPO)) for path in TABLE_PATHS.values()),
        str(INDEX_PATH.relative_to(REPO)),
        str(RECEIPT_PATH.relative_to(REPO)),
        str(MANIFEST_PATH.relative_to(REPO)),
    ]
)

print("Staged files:")
for path in staged:
    print(" ", path)

if sorted(staged) != expected_staged:
    raise RuntimeError(
        "Stage26-8D3 staged-file set differs from the expected 11 files."
    )

git("commit", "-m", COMMIT_MESSAGE)

new_head = git("rev-parse", "HEAD")
parent = git("rev-parse", "HEAD^")
subject = git("log", "-1", "--pretty=%s")

print("Parent :", parent)
print("HEAD   :", new_head)
print("Subject:", subject)

if parent != EXPECTED_PARENT:
    raise RuntimeError("Stage26-8D3 commit parent mismatch.")
if subject != COMMIT_MESSAGE:
    raise RuntimeError("Unexpected Stage26-8D3 commit subject.")


# =============================================================================
# 20. AUTHENTICATED PUSH
# =============================================================================

banner("STAGE26-8D3 :: PUSH")

try:
    from kaggle_secrets import UserSecretsClient
except Exception as exc:
    raise RuntimeError("Kaggle UserSecretsClient unavailable.") from exc

try:
    github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "Could not read Kaggle Secret GITHUB_TOKEN. "
        "Do not paste the token into notebook source."
    ) from exc

if not github_token or len(github_token.strip()) < 20:
    raise RuntimeError("GITHUB_TOKEN secret is empty or unusable.")

askpass_fd, askpass_name = tempfile.mkstemp(
    prefix="stage26_8d3_askpass_",
    suffix=".sh",
)
os.close(askpass_fd)
askpass_path = Path(askpass_name)

try:
    askpass_path.write_text(
        "#!/bin/sh\n"
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *Password*) printf "%s\\n" "$STAGE26_GITHUB_TOKEN" ;;\n'
        '  *) printf "%s\\n" "" ;;\n'
        "esac\n",
        encoding="utf-8",
    )
    askpass_path.chmod(
        askpass_path.stat().st_mode
        | stat.S_IXUSR
        | stat.S_IXGRP
        | stat.S_IXOTH
    )

    push_env = os.environ.copy()
    push_env["GIT_ASKPASS"] = str(askpass_path)
    push_env["GIT_TERMINAL_PROMPT"] = "0"
    push_env["STAGE26_GITHUB_TOKEN"] = github_token.strip()

    push_output = git("push", "origin", "main", env=push_env)
    print(push_output if push_output else "<push completed>")

finally:
    try:
        askpass_path.unlink(missing_ok=True)
    except Exception:
        pass
    github_token = None
    if "push_env" in locals():
        push_env.pop("STAGE26_GITHUB_TOKEN", None)


# =============================================================================
# 21. REMOTE BYTE VERIFICATION
# =============================================================================

banner("STAGE26-8D3 :: REMOTE BYTE VERIFICATION")

git("fetch", "origin", "main")
local_after = git("rev-parse", "HEAD")
remote_after = git("rev-parse", "origin/main")

print("Local HEAD :", local_after)
print("origin/main:", remote_after)

if local_after != new_head or remote_after != new_head:
    raise RuntimeError("Local/remote commit mismatch after Stage26-8D3 push.")

all_output_paths = [
    *TABLE_PATHS.values(),
    INDEX_PATH,
    RECEIPT_PATH,
    MANIFEST_PATH,
]

verification = []

for path in all_output_paths:
    rel = str(path.relative_to(REPO))
    local_bytes = path.read_bytes()
    remote_bytes = git_blob_bytes("origin/main", rel)
    local_sha = sha256_bytes(local_bytes)
    remote_sha = sha256_bytes(remote_bytes)
    same = local_bytes == remote_bytes
    verification.append(
        {
            "path": rel,
            "local_sha256": local_sha,
            "remote_sha256": remote_sha,
            "byte_identical": same,
        }
    )
    print(f"{'PASS' if same else 'FAIL'} {rel}")
    print("  local :", local_sha)
    print("  remote:", remote_sha)
    if not same:
        raise RuntimeError(f"Remote byte verification failed: {rel}")


# =============================================================================
# 22. FINAL CLEAN-REPO CLOSURE
# =============================================================================

final_status = git("status", "--porcelain")

banner("STAGE26-8D3 COMPLETE")

print("Durable table-generation anchor:", new_head)
print("HEAD == origin/main          :", local_after == remote_after == new_head)
print("Repo clean                   :", final_status == "")
print(
    "Remote files byte-identical  :",
    all(item["byte_identical"] for item in verification),
)
print("Publication tables generated :", len(output_records))

print("\nROW COUNTS:")
for record in output_records:
    print(f"  {record['table_id']:50s} {record['row_count']}")

print("\nSCIENTIFIC STATE:")
print("  effective schema            : STAGE26-8D2 + STAGE26-8D2A")
print("  cold-start uncertainty      : POINT ESTIMATES ONLY")
print("  warm uncertainty            : STAGE26-6F1 CORRECTED ONLY")
print("  memory aggregation          : NONE")
print("  Stage26-7C                  : COPIED / NOT RECOMPUTED")
print("  Group-B ratio               : COPIED / NOT INVERTED")
print("  Stage26-6D Pareto           : COPIED / NOT RECOMPUTED")
print("  historical Stage26-4C3      : RETAINED")
print("  Stage26-8 sensitivity       : DESCRIPTIVE / SEPARATE")
print("  complete E2E                : UNAVAILABLE")
print("  timing                      : NO")
print("  inference                   : NO")
print("  bootstrap                   : NO")
print("  PCAP                        : NO")
print("  labels                      : NO")
print("  GPU                         : OFF")

if final_status:
    raise RuntimeError("Repository is not clean after Stage26-8D3.")

print("\nNEXT:")
print("  Generate the eight CPU publication figures from the frozen schema")
print("  and the now-durable Stage26-8D3 publication tables.")
print("  GPU remains OFF.")



STAGE26-8D3 :: DURABLE STATE GATE
Expected parent: 80f7fce76b3a859ea699e8838eced34e9936d421
Local HEAD     : 80f7fce76b3a859ea699e8838eced34e9936d421
origin/main    : 80f7fce76b3a859ea699e8838eced34e9936d421
Repo clean     : True

STAGE26-8D3 :: EFFECTIVE SCHEMA GATE
Expected effective schema SHA256: 973cc58ed6572d30b1cce94f3659226d99b185390c2aca92d9ed030e50358ad0
Actual effective schema SHA256  : 973cc58ed6572d30b1cce94f3659226d99b185390c2aca92d9ed030e50358ad0
Table IDs:
  T26_CPU_WARM_INFERENCE
  T26_CPU_MEMORY_PACKAGE
  T26_CPU_COLD_START
  T26_CPU_CAPACITY_SCALING
  T26_COMPONENT_MEASUREMENTS
  T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO
  T26_PARETO
  T26_REPRESENTATION_SENSITIVITY

STAGE26-8D3 :: SOURCE BYTE-IDENTITY GATES
PASS warm_cpu_point_estimates
  path      : results/stage26_deployment_profiling/stage26_2_cpu_warm_inference/stage26_2_condition_status.csv
  anchor    : 7ffdba3f4ca4ea5cc53097d62aaf27957009b9f6
  historical: df146563826992cb57702e589e4cf5d875ecfdbbf79533a7314

In [20]:
# =============================================================================
# STAGE26-8D4
# GENERATE EIGHT CPU PUBLICATION FIGURES FROM DURABLE STAGE26-8D3 TABLES
#
# SCIENTIFIC PARENT:
#   b11936a4ee8234a36a29057658f7ee420a5cf68e
#
# PURPOSE
# -------
# Generate the eight publication-facing CPU figure families frozen by
# Stage26-8D2 + Stage26-8D2A, using ONLY the durable Stage26-8D3 publication
# tables as quantitative inputs.
#
# THIS CELL DOES NOT:
#   - read raw timing observations
#   - read raw memory observations
#   - read raw PCAP / labels
#   - run inference
#   - load models
#   - run bootstrap
#   - recompute Stage26-7C ratios/scaling
#   - recompute Stage26-6D Pareto membership
#   - reconstruct cold-start confidence intervals
#   - aggregate memory across CPU/batch conditions
#   - impute resource-limit values
#   - use GPU
#
# FIGURE FAMILIES
# ---------------
#   F26_WARM_LATENCY
#   F26_WARM_THROUGHPUT
#   F26_MEMORY_PACKAGE
#   F26_COLD_START
#   F26_CAPACITY_SCALING
#   F26_PARETO
#   F26_REPRESENTATION_SENSITIVITY
#   F26_COMPONENT_BOUNDARY
#
# Each logical figure is written as both PNG (300 dpi) and PDF.
# =============================================================================

from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import stat
import subprocess
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
import numpy as np
import pandas as pd


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")
RESULT_ROOT = REPO / "results" / "stage26_deployment_profiling"

EXPECTED_PARENT = "b11936a4ee8234a36a29057658f7ee420a5cf68e"
EXPECTED_EFFECTIVE_SCHEMA_SHA256 = (
    "973cc58ed6572d30b1cce94f3659226d99b185390c2aca92d9ed030e50358ad0"
)

EFFECTIVE_SCHEMA = (
    RESULT_ROOT
    / "stage26_8d2a_cpu_publication_schema_erratum"
    / "stage26_8d2a_effective_cpu_publication_schema.json"
)

TABLE_DIR = RESULT_ROOT / "stage26_8d3_cpu_publication_tables"

TABLES = {
    "T26_CPU_WARM_INFERENCE": TABLE_DIR / "T26_CPU_WARM_INFERENCE.csv",
    "T26_CPU_MEMORY_PACKAGE": TABLE_DIR / "T26_CPU_MEMORY_PACKAGE.csv",
    "T26_CPU_COLD_START": TABLE_DIR / "T26_CPU_COLD_START.csv",
    "T26_CPU_CAPACITY_SCALING": TABLE_DIR / "T26_CPU_CAPACITY_SCALING.csv",
    "T26_COMPONENT_MEASUREMENTS": TABLE_DIR / "T26_COMPONENT_MEASUREMENTS.csv",
    "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO": (
        TABLE_DIR / "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO.csv"
    ),
    "T26_PARETO": TABLE_DIR / "T26_PARETO.csv",
    "T26_REPRESENTATION_SENSITIVITY": (
        TABLE_DIR / "T26_REPRESENTATION_SENSITIVITY.csv"
    ),
}

EXPECTED_TABLE_SHA256 = {
    "T26_CPU_WARM_INFERENCE": (
        "d664feca94308ea544506df7ef5c997bc4486f457153abfc1959db36808cfaad"
    ),
    "T26_CPU_MEMORY_PACKAGE": (
        "1daa8aa64288bd06fa93ed485dfc696751ef7a0927ef4fe9c8802b4c56c0a458"
    ),
    "T26_CPU_COLD_START": (
        "9d822632c93233e48113795f04c4e646fe0f330f10248fd7e12f8742b1b77803"
    ),
    "T26_CPU_CAPACITY_SCALING": (
        "a5beda3df22ad701f2910e13201ba689788b667f02507cf64739c3bc225fc41e"
    ),
    "T26_COMPONENT_MEASUREMENTS": (
        "dd4dedfe595e2cc29f1eb44c97d66f2c9b8d58d7c5924cafb9129be11ce76d67"
    ),
    "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO": (
        "a29b334c2287edf4e02712bb4dfdc737a326fde4847fd9f1dcfadb7b83299f69"
    ),
    "T26_PARETO": (
        "7d9f7c804bcb99c7d9307689c60442bd3a7c88ea84552c7f49863016705d3200"
    ),
    "T26_REPRESENTATION_SENSITIVITY": (
        "898b386c6a6a1432383386195b611ddadb7eb587698fc095b8df632f9197ee66"
    ),
}

EXPECTED_TABLE_ROWS = {
    "T26_CPU_WARM_INFERENCE": 80,
    "T26_CPU_MEMORY_PACKAGE": 400,
    "T26_CPU_COLD_START": 56,
    "T26_CPU_CAPACITY_SCALING": 40,
    "T26_COMPONENT_MEASUREMENTS": 4,
    "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO": 7,
    "T26_PARETO": 6,
    "T26_REPRESENTATION_SENSITIVITY": 5,
}

OUT_DIR = RESULT_ROOT / "stage26_8d4_cpu_publication_figures"

FIGURE_INDEX_PATH = OUT_DIR / "stage26_8d4_cpu_publication_figures_index.json"
RECEIPT_PATH = OUT_DIR / "stage26_8d4_cpu_publication_figures_receipt.json"
MANIFEST_PATH = OUT_DIR / "stage26_8d4_cpu_publication_figures_manifest.json"

COMMIT_MESSAGE = "stage26: generate CPU publication figures"

BATCHES = [1, 64, 256, 1024, 8192]
CPU_MODES = ["CPU_1_PHYSICAL_CORE", "CPU_2_PHYSICAL_CORE"]
CPU_LABEL = {
    "CPU_1_PHYSICAL_CORE": "CPU1 (1 physical core)",
    "CPU_2_PHYSICAL_CORE": "CPU2 (2 physical cores)",
}

TARGET_ORDER = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
    "ENS_LGBM_XGB_EQUAL",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
]

TARGET_LABEL = {
    "STAGE16_XGBOOST_TUNED": "XGBoost",
    "STAGE16_LIGHTGBM_TUNED": "LightGBM",
    "STAGE16_CATBOOST_TUNED": "CatBoost",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING": "FT ensemble",
    "STAGE20_MASKED_CNN_V1": "CNN",
    "STAGE21_MASKED_VIT_V1": "ViT",
    "ENS_LGBM_XGB_EQUAL": "LGBM+XGB ref",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE": "FT single ref",
}

PRIMARY_TARGETS = {
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
}

MEMORY_METRIC_ORDER = [
    "median_baseline_rss_mib",
    "median_loaded_rss_mib",
    "median_peak_rss_mib",
    "median_delta_model_rss_mib",
    "median_delta_peak_rss_mib",
]

MEMORY_METRIC_LABEL = {
    "median_baseline_rss_mib": "Median baseline RSS (MiB)",
    "median_loaded_rss_mib": "Median loaded RSS (MiB)",
    "median_peak_rss_mib": "Median peak RSS (MiB)",
    "median_delta_model_rss_mib": "Median model ΔRSS (MiB)",
    "median_delta_peak_rss_mib": "Median peak ΔRSS (MiB)",
}

COLD_COMPONENT_LABEL = {
    "process_spawn_to_worker_ready_ns": "spawn→ready",
    "framework_import_ns": "framework import",
    "input_preparation_ns": "input prep",
    "model_deserialization_load_ns": "model load",
    "first_prediction_ns": "first prediction",
    "spawn_to_first_output_ns": "spawn→first output",
    "parent_process_total_ns": "parent total",
}

PUBLICATION_LABEL = (
    "COMPONENT_LEVEL_WITH_STAGE26_8_4C3_SENSITIVITY_AUDIT_COMPLETED"
)

# Stable rendering metadata. No current timestamp is embedded in image metadata.
PNG_METADATA = {
    "Software": "Stage26-8D4 matplotlib publication renderer",
}
PDF_METADATA = {
    "Title": "Stage26 CPU publication figure",
    "Author": "Stage26 validation-safe ablation pipeline",
    "Subject": "CPU component-level deployment profiling",
    "Keywords": "Stage26 CPU component-level",
    "Creator": "Stage26-8D4 matplotlib publication renderer",
    "Producer": "Stage26-8D4 matplotlib publication renderer",
    "CreationDate": None,
    "ModDate": None,
}


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text: str) -> None:
    print("\n" + "=" * 124)
    print(text)
    print("=" * 124)


def run(cmd, *, cwd=REPO, check=True, env=None):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
        env=env,
    )
    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}\n{p.stdout}"
        )
    return p.stdout.strip()


def git(*args, check=True, env=None):
    return run(["git", *args], check=check, env=env)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            block = f.read(8 * 1024 * 1024)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def read_json(path: Path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def atomic_json(path: Path, payload) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, sort_keys=True, allow_nan=False)
        f.write("\n")
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def git_blob_bytes(ref: str, rel_path: str) -> bytes:
    p = subprocess.run(
        ["git", "show", f"{ref}:{rel_path}"],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )
    if p.returncode != 0:
        raise RuntimeError(
            f"Could not read {rel_path} from {ref}:\n"
            + p.stderr.decode("utf-8", errors="replace")
        )
    return p.stdout


def save_figure(fig, figure_id: str):
    png = OUT_DIR / f"{figure_id}.png"
    pdf = OUT_DIR / f"{figure_id}.pdf"

    fig.savefig(
        png,
        dpi=300,
        bbox_inches="tight",
        metadata=PNG_METADATA,
    )
    fig.savefig(
        pdf,
        bbox_inches="tight",
        metadata=PDF_METADATA,
    )
    plt.close(fig)
    return png, pdf


def as_numeric(df: pd.DataFrame, columns):
    for column in columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")
    return df


def assert_columns(df: pd.DataFrame, required, label: str):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"{label} missing columns: {missing}")


def target_sort_key(target_id: str):
    try:
        return TARGET_ORDER.index(target_id)
    except ValueError:
        return 999


def ordered_targets(values):
    return sorted(set(values), key=target_sort_key)


def safe_errorbar(ax, x, y, low, high, *, label=None, marker="o"):
    x = np.asarray(x)
    y = np.asarray(y, dtype=float)
    low = np.asarray(low, dtype=float)
    high = np.asarray(high, dtype=float)
    mask = np.isfinite(y) & np.isfinite(low) & np.isfinite(high)
    if not mask.any():
        return
    lower = np.maximum(0.0, y[mask] - low[mask])
    upper = np.maximum(0.0, high[mask] - y[mask])
    ax.errorbar(
        x[mask],
        y[mask],
        yerr=np.vstack([lower, upper]),
        fmt=marker,
        capsize=2,
        linewidth=1,
        markersize=4,
        label=label,
    )


def annotate_resource_limits(ax, subset: pd.DataFrame, x_lookup):
    nonpass = subset[subset["status"] != "PASS"].copy()
    if nonpass.empty:
        return
    y0, y1 = ax.get_ylim()
    if ax.get_yscale() == "log":
        if y0 <= 0:
            y0 = 1e-6
        y_marker = y0 * ((y1 / y0) ** 0.92)
    else:
        y_marker = y0 + 0.92 * (y1 - y0)
    for _, row in nonpass.iterrows():
        x = x_lookup[int(row["batch_size"])]
        outcome = str(row["resource_limit_outcome"])
        short = "OOM" if "OOM" in outcome else "TO"
        ax.scatter([x], [y_marker], marker="x", s=36)
        ax.annotate(
            short,
            (x, y_marker),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=7,
        )


def format_batch_axis(ax):
    xs = np.arange(len(BATCHES))
    ax.set_xticks(xs)
    ax.set_xticklabels([str(b) for b in BATCHES])
    ax.set_xlabel("Batch size")
    ax.grid(True, axis="y", alpha=0.25)


def figure_record(figure_id, title, caption, png, pdf, sources, validations):
    return {
        "figure_id": figure_id,
        "title": title,
        "caption": caption,
        "source_tables": sources,
        "validations": validations,
        "files": [
            {
                "format": "PNG",
                "path": str(png.relative_to(REPO)),
                "sha256": sha256_file(png),
                "size_bytes": int(png.stat().st_size),
            },
            {
                "format": "PDF",
                "path": str(pdf.relative_to(REPO)),
                "sha256": sha256_file(pdf),
                "size_bytes": int(pdf.stat().st_size),
            },
        ],
    }


# =============================================================================
# 2. DURABLE STATE GATE
# =============================================================================

banner("STAGE26-8D4 :: DURABLE STATE GATE")

git("fetch", "origin", "main")
head = git("rev-parse", "HEAD")
remote = git("rev-parse", "origin/main")
status = git("status", "--porcelain")

print("Expected parent:", EXPECTED_PARENT)
print("Local HEAD     :", head)
print("origin/main    :", remote)
print("Repo clean     :", status == "")

if head != EXPECTED_PARENT:
    raise RuntimeError("Unexpected local HEAD before Stage26-8D4.")
if remote != EXPECTED_PARENT:
    raise RuntimeError("origin/main changed before Stage26-8D4.")
if status:
    raise RuntimeError("Repository must be clean before Stage26-8D4.")
if OUT_DIR.exists():
    raise RuntimeError(
        "Stage26-8D4 output directory already exists. Do not overwrite a durable figure package."
    )


# =============================================================================
# 3. EFFECTIVE SCHEMA + FROZEN FIGURE IDS
# =============================================================================

banner("STAGE26-8D4 :: EFFECTIVE FIGURE SCHEMA GATE")

if not EFFECTIVE_SCHEMA.is_file():
    raise FileNotFoundError(EFFECTIVE_SCHEMA)

schema_sha = sha256_file(EFFECTIVE_SCHEMA)
print("Expected effective schema SHA256:", EXPECTED_EFFECTIVE_SCHEMA_SHA256)
print("Actual effective schema SHA256  :", schema_sha)
if schema_sha != EXPECTED_EFFECTIVE_SCHEMA_SHA256:
    raise RuntimeError("Effective publication schema SHA mismatch.")

schema = read_json(EFFECTIVE_SCHEMA)
figure_specs = schema["figures"]
figure_ids = [item["id"] for item in figure_specs]

EXPECTED_FIGURE_IDS = [
    "F26_WARM_LATENCY",
    "F26_WARM_THROUGHPUT",
    "F26_MEMORY_PACKAGE",
    "F26_COLD_START",
    "F26_CAPACITY_SCALING",
    "F26_PARETO",
    "F26_REPRESENTATION_SENSITIVITY",
    "F26_COMPONENT_BOUNDARY",
]

print("Figure IDs:")
for figure_id in figure_ids:
    print(" ", figure_id)

if figure_ids != EXPECTED_FIGURE_IDS:
    raise RuntimeError("Frozen figure ID/order changed.")
if schema["scope"]["gpu_allowed"] is not False:
    raise RuntimeError("GPU must remain OFF during CPU figure closure.")
if schema["cold_start_publication_policy"] != "COLD_START_POINT_ESTIMATES_ONLY":
    raise RuntimeError("Cold-start publication policy changed.")


# =============================================================================
# 4. VERIFY DURABLE STAGE26-8D3 TABLE IDENTITIES
# =============================================================================

banner("STAGE26-8D4 :: STAGE26-8D3 TABLE IDENTITY GATES")

for table_id in EXPECTED_TABLE_SHA256:
    path = TABLES[table_id]
    if not path.is_file():
        raise FileNotFoundError(path)
    actual_sha = sha256_file(path)
    print(f"{table_id:54s} expected={EXPECTED_TABLE_SHA256[table_id]}")
    print(f"{'':54s} actual  ={actual_sha}")
    if actual_sha != EXPECTED_TABLE_SHA256[table_id]:
        raise RuntimeError(f"Stage26-8D3 table SHA mismatch: {table_id}")


# =============================================================================
# 5. LOAD ONLY PUBLICATION TABLES
# =============================================================================

banner("STAGE26-8D4 :: LOAD PUBLICATION TABLES ONLY")

dfs = {table_id: pd.read_csv(path) for table_id, path in TABLES.items()}
for table_id, df in dfs.items():
    print(f"{table_id:54s} rows={len(df):4d}")
    if len(df) != EXPECTED_TABLE_ROWS[table_id]:
        raise RuntimeError(f"Unexpected row count for {table_id}.")

warm = dfs["T26_CPU_WARM_INFERENCE"].copy()
memory = dfs["T26_CPU_MEMORY_PACKAGE"].copy()
cold = dfs["T26_CPU_COLD_START"].copy()
capacity = dfs["T26_CPU_CAPACITY_SCALING"].copy()
components = dfs["T26_COMPONENT_MEASUREMENTS"].copy()
group_b_ratio = dfs["T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO"].copy()
pareto = dfs["T26_PARETO"].copy()
sensitivity = dfs["T26_REPRESENTATION_SENSITIVITY"].copy()

as_numeric(
    warm,
    [
        "batch_size",
        "n_timed_observations",
        "p50_latency_ms",
        "p50_ci95_low_ms",
        "p50_ci95_high_ms",
        "p95_latency_ms",
        "p95_ci95_low_ms",
        "p95_ci95_high_ms",
        "p99_latency_ms",
        "p99_ci95_low_ms",
        "p99_ci95_high_ms",
        "median_throughput_samples_per_s",
        "throughput_ci95_low_samples_per_s",
        "throughput_ci95_high_samples_per_s",
    ],
)

as_numeric(
    memory,
    [
        "batch_size",
        "deployment_package_size_mib",
        "memory_value_mib",
        "planned_repetitions",
        "pass_repetitions",
        "resource_limit_oom_repetitions",
        "timeout_resource_limit_repetitions",
    ],
)

as_numeric(cold, ["n_observations", "p50_ms", "p95_ms"])
as_numeric(
    capacity,
    [
        "batch_size",
        "cpu1_throughput_multiplier_vs_b1",
        "cpu2_throughput_multiplier_vs_b1",
        "cpu2_over_cpu1_speedup",
        "physical_core_parallel_efficiency",
    ],
)
as_numeric(group_b_ratio, ["batch_size", "inference_over_representation_capacity_ratio"])
as_numeric(pareto, ["pr_auc", "cpu1_batch1_p95_latency_ms"])
as_numeric(
    sensitivity,
    ["batch_size", "n_pairs", "fingerprints_equal_count", "median_throughput_ratio_v3_over_v2"],
)


# =============================================================================
# 6. PUBLICATION TABLE SAFETY GATES BEFORE PLOTTING
# =============================================================================

banner("STAGE26-8D4 :: PUBLICATION TABLE SAFETY GATES")

# Warm: exact 73 PASS / 7 resource limits.
if int((warm["status"] == "PASS").sum()) != 73:
    raise RuntimeError("Warm PASS count changed.")
if int((warm["status"] != "PASS").sum()) != 7:
    raise RuntimeError("Warm resource-limit count changed.")

for _, row in warm[warm["status"] != "PASS"].iterrows():
    quantitative = [
        "p50_latency_ms",
        "p50_ci95_low_ms",
        "p50_ci95_high_ms",
        "p95_latency_ms",
        "p95_ci95_low_ms",
        "p95_ci95_high_ms",
        "p99_latency_ms",
        "p99_ci95_low_ms",
        "p99_ci95_high_ms",
        "median_throughput_samples_per_s",
        "throughput_ci95_low_samples_per_s",
        "throughput_ci95_high_samples_per_s",
    ]
    if any(pd.notna(row[c]) for c in quantitative):
        raise RuntimeError("Non-PASS warm row contains quantitative values.")

# Cold-start CIs must not exist at all.
if any("ci" in c.lower() for c in cold.columns):
    raise RuntimeError("Cold-start publication table contains CI columns.")

# Memory must remain condition-resolved: 8 targets x 2 CPU x 5 batch x 5 metrics.
if set(memory["memory_metric_name"].astype(str)) != set(MEMORY_METRIC_ORDER):
    raise RuntimeError("Memory metric universe changed.")
if len(memory) != 8 * 2 * 5 * 5:
    raise RuntimeError("Memory table is not condition-resolved as frozen.")

# Group-B capacity ratio direction/label fixed by 8D2A.
if set(group_b_ratio["publication_label"].astype(str)) != {PUBLICATION_LABEL}:
    raise RuntimeError("Group-B publication label changed.")
if (group_b_ratio["inference_over_representation_capacity_ratio"] <= 0).any():
    raise RuntimeError("Invalid Group-B component-capacity ratio.")

# Pareto universe/membership is copied, not derived here.
if len(pareto) != 6:
    raise RuntimeError("Pareto row count changed.")
if set(pareto["group"].astype(str)) != {"GROUP_A_DUPSAFE70", "GROUP_B_PACKET_IMAGE"}:
    raise RuntimeError("Pareto group universe changed.")

# Sensitivity remains exactly five complete paired batches.
if sensitivity["batch_size"].astype(int).tolist() != BATCHES:
    raise RuntimeError("Sensitivity batch order/universe changed.")
if not (sensitivity["n_pairs"] == 5).all():
    raise RuntimeError("Sensitivity pair count changed.")
if not (sensitivity["fingerprints_equal_count"] == 5).all():
    raise RuntimeError("Sensitivity fingerprint-equivalence count changed.")

# Component boundary must explicitly close complete E2E.
e2e = components[components["component_family"] == "COMPLETE_E2E"]
if len(e2e) != 1:
    raise RuntimeError("Expected exactly one complete-E2E boundary row.")
if str(e2e.iloc[0]["measurement_status"]) != "CLOSED_NO_VALID_COMPLETE_E2E_MEASUREMENT":
    raise RuntimeError("Complete-E2E closure changed.")
if bool(e2e.iloc[0]["complete_e2e_available"]):
    raise RuntimeError("Complete E2E must remain unavailable.")

print("Warm resource-limit preservation : PASS")
print("Cold-start CI exclusion          : PASS")
print("Memory condition resolution      : PASS")
print("Group-B ratio direction          : PASS")
print("Pareto membership source-only    : PASS")
print("Sensitivity paired geometry      : PASS")
print("Complete E2E unavailable         : PASS")
print("GPU                              : OFF")


# =============================================================================
# 7. CREATE OUTPUT DIRECTORY
# =============================================================================

OUT_DIR.mkdir(parents=True, exist_ok=False)

# Keep rcParams local/simple; do not use external style packages.
plt.rcParams.update(
    {
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "legend.fontsize": 7,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "figure.titlesize": 12,
    }
)

figure_records = []


# =============================================================================
# 8. F26_WARM_LATENCY
# =============================================================================

banner("STAGE26-8D4 :: F26_WARM_LATENCY")

fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True)
metric_specs = [
    ("p50_latency_ms", "p50_ci95_low_ms", "p50_ci95_high_ms", "p50 latency"),
    ("p95_latency_ms", "p95_ci95_low_ms", "p95_ci95_high_ms", "p95 latency"),
]
x_lookup = {b: i for i, b in enumerate(BATCHES)}

for row_idx, cpu_mode in enumerate(CPU_MODES):
    cpu_df = warm[warm["cpu_mode"] == cpu_mode].copy()
    for col_idx, (value_col, lo_col, hi_col, metric_label) in enumerate(metric_specs):
        ax = axes[row_idx, col_idx]
        for target in ordered_targets(cpu_df["target_id"]):
            sub = cpu_df[cpu_df["target_id"] == target].copy()
            sub = sub[sub["status"] == "PASS"].sort_values("batch_size")
            if sub.empty:
                continue
            xs = np.array([x_lookup[int(v)] for v in sub["batch_size"]])
            safe_errorbar(
                ax,
                xs,
                sub[value_col].to_numpy(),
                sub[lo_col].to_numpy(),
                sub[hi_col].to_numpy(),
                label=TARGET_LABEL.get(target, target),
            )

        ax.set_yscale("log")

        # Resource-limit outcomes are annotated from Stage26-8D3; no interpolation.
        for target in ordered_targets(cpu_df["target_id"]):
            sub_all = cpu_df[cpu_df["target_id"] == target]
            annotate_resource_limits(ax, sub_all, x_lookup)

        format_batch_axis(ax)
        ax.set_ylabel("Latency (ms, log scale)")
        ax.set_title(f"{CPU_LABEL[cpu_mode]} — {metric_label}")

handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False)
fig.suptitle("CPU isolated warm-inference latency by batch size")
fig.text(
    0.5,
    0.02,
    "Points are Stage26-2 estimates; error bars are corrected Stage26-6F1 95% intervals. "
    "X annotations preserve OOM/timeout outcomes; no line interpolation is used.",
    ha="center",
    fontsize=8,
)
fig.subplots_adjust(bottom=0.13, hspace=0.28, wspace=0.22)
png, pdf = save_figure(fig, "F26_WARM_LATENCY")
figure_records.append(
    figure_record(
        "F26_WARM_LATENCY",
        "CPU isolated warm-inference latency by batch size",
        "Stage26-2 p50/p95 point estimates with corrected Stage26-6F1 uncertainty; CPU1 and CPU2 are separate panels and resource-limit outcomes are explicit.",
        png,
        pdf,
        ["T26_CPU_WARM_INFERENCE"],
        [
            "CORRECTED_STAGE26_6F1_UNCERTAINTY_ONLY",
            "NO_STAGE26_2_HISTORICAL_CI",
            "RESOURCE_LIMITS_EXPLICIT",
            "NO_INTERPOLATION_THROUGH_RESOURCE_LIMITS",
        ],
    )
)
print("PASS F26_WARM_LATENCY")


# =============================================================================
# 9. F26_WARM_THROUGHPUT
# =============================================================================

banner("STAGE26-8D4 :: F26_WARM_THROUGHPUT")

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=True)
for ax, cpu_mode in zip(axes, CPU_MODES):
    cpu_df = warm[warm["cpu_mode"] == cpu_mode].copy()
    for target in ordered_targets(cpu_df["target_id"]):
        sub = cpu_df[(cpu_df["target_id"] == target) & (cpu_df["status"] == "PASS")]
        sub = sub.sort_values("batch_size")
        if sub.empty:
            continue
        xs = np.array([x_lookup[int(v)] for v in sub["batch_size"]])
        safe_errorbar(
            ax,
            xs,
            sub["median_throughput_samples_per_s"].to_numpy(),
            sub["throughput_ci95_low_samples_per_s"].to_numpy(),
            sub["throughput_ci95_high_samples_per_s"].to_numpy(),
            label=TARGET_LABEL.get(target, target),
        )

    ax.set_yscale("log")

    for target in ordered_targets(cpu_df["target_id"]):
        annotate_resource_limits(ax, cpu_df[cpu_df["target_id"] == target], x_lookup)

    format_batch_axis(ax)
    ax.set_ylabel("Median throughput (samples/s, log scale)")
    ax.set_title(CPU_LABEL[cpu_mode])

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False)
fig.suptitle("CPU isolated warm-inference throughput by batch size")
fig.text(
    0.5,
    0.03,
    "Stage26-2 median throughput with corrected Stage26-6F1 95% intervals; no post-hoc best-batch highlighting.",
    ha="center",
    fontsize=8,
)
fig.subplots_adjust(bottom=0.18, wspace=0.22)
png, pdf = save_figure(fig, "F26_WARM_THROUGHPUT")
figure_records.append(
    figure_record(
        "F26_WARM_THROUGHPUT",
        "CPU isolated warm-inference throughput by batch size",
        "Median isolated-inference throughput with corrected Stage26-6F1 uncertainty, separate CPU1/CPU2 panels, and explicit OOM/timeout preservation.",
        png,
        pdf,
        ["T26_CPU_WARM_INFERENCE"],
        [
            "CORRECTED_STAGE26_6F1_UNCERTAINTY_ONLY",
            "RESOURCE_LIMITS_EXPLICIT",
            "NO_POST_HOC_BEST_BATCH",
        ],
    )
)
print("PASS F26_WARM_THROUGHPUT")


# =============================================================================
# 10. F26_MEMORY_PACKAGE
# =============================================================================

banner("STAGE26-8D4 :: F26_MEMORY_PACKAGE")

# Package size is displayed in its own panel. Runtime memory is shown as five
# condition-resolved heatmaps (all five metrics frozen by 8D2A), with no
# aggregation across CPU mode or batch size.
fig = plt.figure(figsize=(20, 26))
gs = fig.add_gridspec(6, 1, height_ratios=[1.0, 1.2, 1.2, 1.2, 1.2, 1.2], hspace=0.48)

# Panel 1: immutable deployment package size per target.
ax_pkg = fig.add_subplot(gs[0, 0])
pkg = (
    memory[["target_id", "deployment_package_size_mib"]]
    .drop_duplicates()
    .copy()
)
if len(pkg) != 8:
    raise RuntimeError("Expected exactly eight target package sizes.")
pkg["_order"] = pkg["target_id"].map({t: i for i, t in enumerate(TARGET_ORDER)})
pkg = pkg.sort_values("_order")
xs = np.arange(len(pkg))
ax_pkg.bar(xs, pkg["deployment_package_size_mib"].to_numpy())
ax_pkg.set_xticks(xs)
ax_pkg.set_xticklabels(
    [TARGET_LABEL.get(t, t) for t in pkg["target_id"]],
    rotation=25,
    ha="right",
)
ax_pkg.set_ylabel("Deployment package size (MiB)")
ax_pkg.set_title("Deployment package size — independent serialized artifact footprint")
ax_pkg.grid(True, axis="y", alpha=0.25)

# Heatmap columns preserve CPU x batch exactly.
condition_columns = [
    ("CPU_1_PHYSICAL_CORE", 1),
    ("CPU_1_PHYSICAL_CORE", 64),
    ("CPU_1_PHYSICAL_CORE", 256),
    ("CPU_1_PHYSICAL_CORE", 1024),
    ("CPU_1_PHYSICAL_CORE", 8192),
    ("CPU_2_PHYSICAL_CORE", 1),
    ("CPU_2_PHYSICAL_CORE", 64),
    ("CPU_2_PHYSICAL_CORE", 256),
    ("CPU_2_PHYSICAL_CORE", 1024),
    ("CPU_2_PHYSICAL_CORE", 8192),
]
condition_labels = [
    "CPU1\nB1",
    "CPU1\nB64",
    "CPU1\nB256",
    "CPU1\nB1024",
    "CPU1\nB8192",
    "CPU2\nB1",
    "CPU2\nB64",
    "CPU2\nB256",
    "CPU2\nB1024",
    "CPU2\nB8192",
]

for idx, metric in enumerate(MEMORY_METRIC_ORDER, start=1):
    ax = fig.add_subplot(gs[idx, 0])
    arr = np.full((len(TARGET_ORDER), len(condition_columns)), np.nan, dtype=float)
    status_arr = np.empty(arr.shape, dtype=object)
    status_arr[:] = "MISSING"

    metric_df = memory[memory["memory_metric_name"] == metric].copy()
    if len(metric_df) != 80:
        raise RuntimeError(f"Expected 80 condition rows for memory metric {metric}.")

    for i, target in enumerate(TARGET_ORDER):
        for j, (cpu_mode, batch) in enumerate(condition_columns):
            sub = metric_df[
                (metric_df["target_id"] == target)
                & (metric_df["cpu_mode"] == cpu_mode)
                & (metric_df["batch_size"] == batch)
            ]
            if len(sub) != 1:
                raise RuntimeError(
                    f"Expected one memory cell for {target}/{cpu_mode}/B{batch}/{metric}."
                )
            row = sub.iloc[0]
            status_arr[i, j] = str(row["memory_status"])
            if str(row["memory_status"]) == "PASS":
                arr[i, j] = float(row["memory_value_mib"])
            else:
                if pd.notna(row["memory_value_mib"]):
                    raise RuntimeError("Resource-limit memory cell contains imputed value.")

    masked = np.ma.masked_invalid(arr)
    im = ax.imshow(masked, aspect="auto")
    ax.set_xticks(np.arange(len(condition_columns)))
    ax.set_xticklabels(condition_labels)
    ax.set_yticks(np.arange(len(TARGET_ORDER)))
    ax.set_yticklabels([TARGET_LABEL[t] for t in TARGET_ORDER])
    ax.set_title(MEMORY_METRIC_LABEL[metric])
    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label("MiB")

    # Explicitly annotate every unavailable RAM-limit cell.
    for i in range(status_arr.shape[0]):
        for j in range(status_arr.shape[1]):
            if status_arr[i, j] != "PASS":
                text = "OOM" if "OOM" in status_arr[i, j] else "LIMIT"
                ax.text(j, i, text, ha="center", va="center", fontsize=7)

fig.suptitle("Deployment package size and CPU memory profile", y=0.995)
fig.text(
    0.5,
    0.004,
    "Runtime-memory heatmaps preserve every target × CPU mode × batch condition and all five frozen primary RSS metrics; no condition aggregation or imputation is performed.",
    ha="center",
    fontsize=8,
)
png, pdf = save_figure(fig, "F26_MEMORY_PACKAGE")
figure_records.append(
    figure_record(
        "F26_MEMORY_PACKAGE",
        "Deployment package size and CPU memory profile",
        "Serialized package size is shown independently. Runtime memory preserves all five frozen Stage26-3B primary metrics at exact target × CPU × batch resolution, with RAM-limit outcomes annotated rather than imputed.",
        png,
        pdf,
        ["T26_CPU_MEMORY_PACKAGE"],
        [
            "PACKAGE_SIZE_INDEPENDENT_PANEL",
            "ALL_FIVE_FROZEN_MEMORY_METRICS",
            "NO_CPU_OR_BATCH_AGGREGATION",
            "RAM_LIMITS_EXPLICIT",
            "NO_IMPUTATION",
        ],
    )
)
print("PASS F26_MEMORY_PACKAGE")


# =============================================================================
# 11. F26_COLD_START
# =============================================================================

banner("STAGE26-8D4 :: F26_COLD_START")

components_order = list(dict.fromkeys(cold["component"].astype(str).tolist()))
if len(components_order) != 7:
    raise RuntimeError("Expected seven cold-start components.")

fig, axes = plt.subplots(1, 2, figsize=(17, 7), sharex=True)
for ax, value_col, label in [
    (axes[0], "p50_ms", "p50"),
    (axes[1], "p95_ms", "p95"),
]:
    for target in ordered_targets(cold["target_id"]):
        sub = cold[cold["target_id"] == target].copy()
        sub["_order"] = sub["component"].map({c: i for i, c in enumerate(components_order)})
        sub = sub.sort_values("_order")
        xs = sub["_order"].to_numpy(dtype=int)
        ys = sub[value_col].to_numpy(dtype=float)
        ax.scatter(xs, ys, s=24, label=TARGET_LABEL.get(target, target))
    ax.set_yscale("log")
    ax.set_xticks(np.arange(len(components_order)))
    ax.set_xticklabels(
        [COLD_COMPONENT_LABEL.get(c, c) for c in components_order],
        rotation=30,
        ha="right",
    )
    ax.set_ylabel("Cold-start component latency (ms, log scale)")
    ax.set_title(f"{label} point estimates")
    ax.grid(True, axis="y", alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False)
fig.suptitle("CPU cold-start point estimates")
fig.text(
    0.5,
    0.02,
    "Point estimates only. Stage26-1 cold-start confidence intervals are not publication-certified and are not plotted.",
    ha="center",
    fontsize=8,
)
fig.subplots_adjust(bottom=0.25, wspace=0.22)
png, pdf = save_figure(fig, "F26_COLD_START")
figure_records.append(
    figure_record(
        "F26_COLD_START",
        "CPU cold-start point estimates",
        "Cold-start p50 and p95 component point estimates only; no cold-start confidence interval is publication-facing.",
        png,
        pdf,
        ["T26_CPU_COLD_START"],
        ["POINT_ESTIMATES_ONLY", "NO_COLD_START_CI"],
    )
)
print("PASS F26_COLD_START")


# =============================================================================
# 12. F26_CAPACITY_SCALING
# =============================================================================

banner("STAGE26-8D4 :: F26_CAPACITY_SCALING")

fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True)
panels = [
    ("cpu1_throughput_multiplier_vs_b1", "CPU1 throughput multiplier vs B1"),
    ("cpu2_throughput_multiplier_vs_b1", "CPU2 throughput multiplier vs B1"),
    ("cpu2_over_cpu1_speedup", "CPU2 / CPU1 speedup"),
    ("physical_core_parallel_efficiency", "Physical-core parallel efficiency"),
]

for ax, (column, title) in zip(axes.ravel(), panels):
    for target in ordered_targets(capacity["target_id"]):
        sub = capacity[capacity["target_id"] == target].sort_values("batch_size")
        valid = sub[pd.notna(sub[column])]
        if valid.empty:
            continue
        xs = np.array([x_lookup[int(v)] for v in valid["batch_size"]])
        ys = valid[column].to_numpy(dtype=float)
        ax.scatter(xs, ys, s=24, label=TARGET_LABEL.get(target, target))
    format_batch_axis(ax)
    ax.set_title(title)
    ax.set_ylabel("Descriptive scaling ratio")

handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False)
fig.suptitle("CPU component-level scaling")
fig.text(
    0.5,
    0.02,
    "Immutable Stage26-7C descriptive values copied from the publication table; no ratio CI, no post-hoc best-batch selection, and no pipeline-level claim.",
    ha="center",
    fontsize=8,
)
fig.subplots_adjust(bottom=0.13, hspace=0.28, wspace=0.22)
png, pdf = save_figure(fig, "F26_CAPACITY_SCALING")
figure_records.append(
    figure_record(
        "F26_CAPACITY_SCALING",
        "CPU component-level scaling",
        "Within-target Stage26-7C batch scaling and matched CPU2/CPU1 scaling, copied without recomputation or uncertainty propagation.",
        png,
        pdf,
        ["T26_CPU_CAPACITY_SCALING"],
        [
            "STAGE26_7C_COPIED_NOT_RECOMPUTED",
            "NO_RATIO_CI",
            "NO_POST_HOC_BEST_BATCH",
            "COMPONENT_LEVEL_ONLY",
        ],
    )
)
print("PASS F26_CAPACITY_SCALING")


# =============================================================================
# 13. F26_PARETO
# =============================================================================

banner("STAGE26-8D4 :: F26_PARETO")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, group in zip(axes, ["GROUP_A_DUPSAFE70", "GROUP_B_PACKET_IMAGE"]):
    sub = pareto[pareto["group"] == group].copy()
    if sub.empty:
        raise RuntimeError(f"Missing Pareto group: {group}")
    for _, row in sub.iterrows():
        is_frontier = str(row["pareto_status"]) == "FRONTIER_MEMBER"
        marker = "^" if is_frontier else "o"
        facecolors = "none" if not is_frontier else None
        kwargs = {}
        if facecolors is not None:
            kwargs["facecolors"] = facecolors
        ax.scatter(
            [row["cpu1_batch1_p95_latency_ms"]],
            [row["pr_auc"]],
            marker=marker,
            s=70,
            **kwargs,
        )
        ax.annotate(
            TARGET_LABEL.get(row["target_id"], row["target_id"]),
            (row["cpu1_batch1_p95_latency_ms"], row["pr_auc"]),
            xytext=(5, 4),
            textcoords="offset points",
            fontsize=8,
        )
    ax.set_xscale("log")
    ax.set_xlabel("CPU1 B1 p95 isolated-inference latency (ms, log scale)")
    ax.set_ylabel("PR-AUC")
    ax.grid(True, alpha=0.25)
    ax.set_title("Group A" if group == "GROUP_A_DUPSAFE70" else "Group B")

legend_handles = [
    plt.Line2D([0], [0], marker="^", linestyle="None", label="Frontier member"),
    plt.Line2D([0], [0], marker="o", linestyle="None", markerfacecolor="none", label="Not frontier member"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=2, frameon=False)
fig.suptitle("Within-group descriptive point-estimate Pareto frontiers")
fig.text(
    0.5,
    0.02,
    "Group A and Group B are displayed separately. Membership is copied from immutable Stage26-6D; no cross-group frontier and no PR-AUC bootstrap are constructed.",
    ha="center",
    fontsize=8,
)
fig.subplots_adjust(bottom=0.17, wspace=0.25)
png, pdf = save_figure(fig, "F26_PARETO")
figure_records.append(
    figure_record(
        "F26_PARETO",
        "Within-group descriptive point-estimate Pareto frontiers",
        "Stage26-6D point-estimate membership displayed separately for Group A and Group B; no Pareto recomputation, cross-group frontier, or PR-AUC bootstrap.",
        png,
        pdf,
        ["T26_PARETO"],
        [
            "GROUPS_SEPARATE",
            "STAGE26_6D_MEMBERSHIP_COPIED",
            "NO_CROSS_GROUP_FRONTIER",
            "NO_PR_AUC_BOOTSTRAP",
        ],
    )
)
print("PASS F26_PARETO")


# =============================================================================
# 14. F26_REPRESENTATION_SENSITIVITY
# =============================================================================

banner("STAGE26-8D4 :: F26_REPRESENTATION_SENSITIVITY")

fig, ax = plt.subplots(figsize=(10, 5.5))
xs = np.arange(len(BATCHES))
ys = sensitivity["median_throughput_ratio_v3_over_v2"].to_numpy(dtype=float)
ax.plot(xs, ys, marker="o", linewidth=1)
ax.axhline(1.0, linestyle="--", linewidth=1)
ax.set_xticks(xs)
ax.set_xticklabels([str(b) for b in BATCHES])
ax.set_xlabel("Frozen batch size")
ax.set_ylabel("Median throughput ratio V3 / V2")
ax.set_title("Stage26-4C3 paired implementation sensitivity")
ax.grid(True, axis="y", alpha=0.25)
for x, y in zip(xs, ys):
    ax.annotate(f"{y:.3f}", (x, y), xytext=(0, 6), textcoords="offset points", ha="center", fontsize=8)
fig.text(
    0.5,
    0.02,
    "Descriptive paired sensitivity only. The ratio-1 line is a neutral reference; historical Stage26-4C3 remains the original measurement and V3 is not a corrected replacement.",
    ha="center",
    fontsize=8,
)
fig.subplots_adjust(bottom=0.17)
png, pdf = save_figure(fig, "F26_REPRESENTATION_SENSITIVITY")
figure_records.append(
    figure_record(
        "F26_REPRESENTATION_SENSITIVITY",
        "Stage26-4C3 paired implementation sensitivity",
        "Median V3/V2 throughput ratios for the five frozen paired batches, with a neutral ratio-1 reference. Descriptive only; historical 4C3 is retained.",
        png,
        pdf,
        ["T26_REPRESENTATION_SENSITIVITY"],
        [
            "DESCRIPTIVE_ONLY",
            "NEUTRAL_RATIO_ONE_REFERENCE",
            "NO_SIGNIFICANCE_TEST",
            "NO_FIXED_BIAS_CLAIM",
            "V3_NOT_REPLACEMENT",
        ],
    )
)
print("PASS F26_REPRESENTATION_SENSITIVITY")


# =============================================================================
# 15. F26_COMPONENT_BOUNDARY
# =============================================================================

banner("STAGE26-8D4 :: F26_COMPONENT_BOUNDARY")

# This is an inventory/boundary diagram, deliberately without arrows joining
# measured components into a synthetic pipeline. This avoids implying additive
# latency, complete-pipeline throughput, or a full-pipeline bottleneck.
fig, ax = plt.subplots(figsize=(15, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis("off")

rows = components.copy()
row_map = {str(r["component_family"]): r for _, r in rows.iterrows()}

boxes = [
    (
        "RAW_EXTRACTION_COMPONENT",
        0.6,
        6.3,
        2.6,
        2.2,
        "Raw extraction\ncomponent",
    ),
    (
        "PACKET_IMAGE_REPRESENTATION_COMPONENT",
        3.7,
        6.3,
        2.6,
        2.2,
        "Packet-image\nrepresentation",
    ),
    (
        "ISOLATED_WARM_INFERENCE_COMPONENT",
        6.8,
        6.3,
        2.6,
        2.2,
        "Isolated warm\ninference",
    ),
    (
        "COMPLETE_E2E",
        2.2,
        1.6,
        5.6,
        2.4,
        "Complete E2E measurement\nUNAVAILABLE",
    ),
]

for component_family, x, y, w, h, headline in boxes:
    row = row_map[component_family]
    rect = Rectangle((x, y), w, h, fill=False, linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w / 2, y + h * 0.72, headline, ha="center", va="center", fontsize=11)
    ax.text(
        x + w / 2,
        y + h * 0.31,
        f"{row['measurement_status']}\n{row['source_stage']}",
        ha="center",
        va="center",
        fontsize=8,
    )

ax.text(
    5,
    5.2,
    "Distinct measured components — no additive latency or synthesized pipeline throughput",
    ha="center",
    va="center",
    fontsize=10,
)
ax.text(
    5,
    0.65,
    "Complete E2E remains closed because the required representation/extraction bridges are not jointly measured. Missing deployment cost is not imputed.",
    ha="center",
    va="center",
    fontsize=9,
    wrap=True,
)
ax.set_title("Measured deployment component boundaries", pad=12)

png, pdf = save_figure(fig, "F26_COMPONENT_BOUNDARY")
figure_records.append(
    figure_record(
        "F26_COMPONENT_BOUNDARY",
        "Measured deployment component boundaries",
        "Raw extraction, packet-image representation, and isolated inference are shown as distinct measured components; complete E2E is explicitly unavailable and no component values are added into a synthetic pipeline.",
        png,
        pdf,
        ["T26_COMPONENT_MEASUREMENTS"],
        [
            "DISTINCT_COMPONENTS",
            "COMPLETE_E2E_UNAVAILABLE",
            "NO_ADDITIVE_LATENCY",
            "NO_SYNTHETIC_PIPELINE_THROUGHPUT",
            "NO_MISSING_COST_IMPUTATION",
        ],
    )
)
print("PASS F26_COMPONENT_BOUNDARY")


# =============================================================================
# 16. LOGICAL FIGURE COUNT / FILE VALIDATION
# =============================================================================

banner("STAGE26-8D4 :: FIGURE PACKAGE VALIDATION")

if len(figure_records) != 8:
    raise RuntimeError("Expected exactly eight logical publication figures.")
if [r["figure_id"] for r in figure_records] != EXPECTED_FIGURE_IDS:
    raise RuntimeError("Generated figure order differs from frozen schema.")

for record in figure_records:
    if len(record["files"]) != 2:
        raise RuntimeError("Each logical figure must have PNG and PDF output.")
    for item in record["files"]:
        path = REPO / item["path"]
        if not path.is_file() or path.stat().st_size <= 0:
            raise RuntimeError(f"Missing/empty figure file: {path}")

# No figure-index caption/title may contain frozen prohibited wording.
prohibited_exact = set(schema.get("prohibited_wording", []))
for record in figure_records:
    haystack = (record["title"] + " " + record["caption"]).lower()
    for phrase in prohibited_exact:
        if phrase.lower() in haystack:
            raise RuntimeError(
                f"Prohibited publication wording found in figure metadata: {phrase!r}"
            )

print("Logical figures generated :", len(figure_records))
print("Rendered files generated  :", sum(len(r["files"]) for r in figure_records))
print("Formats per figure        : PNG + PDF")
print("Cold-start CIs            : ABSENT")
print("Warm uncertainty          : STAGE26-6F1 corrected only")
print("Memory aggregation        : NONE")
print("Stage26-7C recomputation  : NONE")
print("Pareto recomputation      : NONE")
print("Complete E2E              : UNAVAILABLE")
print("GPU                       : OFF")

for record in figure_records:
    print(f"\n{record['figure_id']}")
    for item in record["files"]:
        print(
            f"  {item['format']:3s} "
            f"SHA256={item['sha256']} "
            f"bytes={item['size_bytes']}"
        )


# =============================================================================
# 17. WRITE INDEX / RECEIPT / MANIFEST
# =============================================================================

banner("STAGE26-8D4 :: WRITE INDEX / RECEIPT / MANIFEST")

figure_index = {
    "schema": "stage26_8d4_cpu_publication_figures_index_v1",
    "scientific_parent": EXPECTED_PARENT,
    "effective_schema_sha256": EXPECTED_EFFECTIVE_SCHEMA_SHA256,
    "publication_label": PUBLICATION_LABEL,
    "logical_figure_count": 8,
    "rendered_file_count": 16,
    "quantitative_input_policy": "STAGE26_8D3_PUBLICATION_TABLES_ONLY",
    "figures": figure_records,
}
atomic_json(FIGURE_INDEX_PATH, figure_index)
figure_index_sha = sha256_file(FIGURE_INDEX_PATH)

receipt = {
    "schema": "stage26_8d4_cpu_publication_figures_receipt_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "scientific_parent": EXPECTED_PARENT,
    "effective_schema_sha256": EXPECTED_EFFECTIVE_SCHEMA_SHA256,
    "table_sha256": EXPECTED_TABLE_SHA256,
    "logical_figure_count": 8,
    "rendered_file_count": 16,
    "figure_ids": EXPECTED_FIGURE_IDS,
    "quantitative_inputs": "STAGE26_8D3_PUBLICATION_TABLES_ONLY",
    "raw_measurement_files_read": False,
    "timing_executed": False,
    "inference_executed": False,
    "model_loaded": False,
    "bootstrap_executed": False,
    "random_sampling_executed": False,
    "memory_condition_aggregation": False,
    "ratio_recomputed": False,
    "ratio_inverted": False,
    "stage26_7c_recomputed": False,
    "pareto_recomputed": False,
    "cold_start_ci_rendered": False,
    "stage26_2_historical_ci_rendered": False,
    "complete_e2e_measurement_available": False,
    "pcap_accessed": False,
    "labels_accessed": False,
    "gpu_used": False,
}
atomic_json(RECEIPT_PATH, receipt)
receipt_sha = sha256_file(RECEIPT_PATH)

manifest_files = []
for record in figure_records:
    for item in record["files"]:
        manifest_files.append(
            {
                "path": item["path"],
                "sha256": item["sha256"],
                "size_bytes": item["size_bytes"],
            }
        )
manifest_files.extend(
    [
        {
            "path": str(FIGURE_INDEX_PATH.relative_to(REPO)),
            "sha256": figure_index_sha,
            "size_bytes": int(FIGURE_INDEX_PATH.stat().st_size),
        },
        {
            "path": str(RECEIPT_PATH.relative_to(REPO)),
            "sha256": receipt_sha,
            "size_bytes": int(RECEIPT_PATH.stat().st_size),
        },
    ]
)

manifest = {
    "schema": "stage26_8d4_cpu_publication_figures_manifest_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "scientific_parent": EXPECTED_PARENT,
    "files": manifest_files,
}
atomic_json(MANIFEST_PATH, manifest)
manifest_sha = sha256_file(MANIFEST_PATH)

print("Figure index SHA256:", figure_index_sha)
print("Receipt SHA256     :", receipt_sha)
print("Manifest SHA256    :", manifest_sha)


# =============================================================================
# 18. PRE-COMMIT GIT AUDIT
# =============================================================================

banner("STAGE26-8D4 :: PRE-COMMIT GIT AUDIT")

status_lines = [
    line for line in git("status", "--porcelain").splitlines() if line.strip()
]
for line in status_lines:
    print(line)

expected_prefix = "?? " + str(OUT_DIR.relative_to(REPO))
if not status_lines:
    raise RuntimeError("No Stage26-8D4 files detected by Git.")
if any(not line.startswith(expected_prefix) for line in status_lines):
    raise RuntimeError("Unexpected repository modification before Stage26-8D4 commit.")


# =============================================================================
# 19. COMMIT
# =============================================================================

banner("STAGE26-8D4 :: COMMIT")

git("add", str(OUT_DIR.relative_to(REPO)))
staged = git("diff", "--cached", "--name-only").splitlines()

expected_paths = []
for figure_id in EXPECTED_FIGURE_IDS:
    expected_paths.extend(
        [
            str((OUT_DIR / f"{figure_id}.png").relative_to(REPO)),
            str((OUT_DIR / f"{figure_id}.pdf").relative_to(REPO)),
        ]
    )
expected_paths.extend(
    [
        str(FIGURE_INDEX_PATH.relative_to(REPO)),
        str(RECEIPT_PATH.relative_to(REPO)),
        str(MANIFEST_PATH.relative_to(REPO)),
    ]
)

print("Staged files:")
for path in staged:
    print(" ", path)

if sorted(staged) != sorted(expected_paths):
    raise RuntimeError(
        f"Stage26-8D4 staged-file set differs from expected package. "
        f"Expected {len(expected_paths)}, got {len(staged)}."
    )

git("commit", "-m", COMMIT_MESSAGE)
new_head = git("rev-parse", "HEAD")
parent = git("rev-parse", "HEAD^")
subject = git("log", "-1", "--pretty=%s")

print("Parent :", parent)
print("HEAD   :", new_head)
print("Subject:", subject)

if parent != EXPECTED_PARENT:
    raise RuntimeError("Stage26-8D4 commit parent mismatch.")
if subject != COMMIT_MESSAGE:
    raise RuntimeError("Unexpected Stage26-8D4 commit subject.")


# =============================================================================
# 20. AUTHENTICATED PUSH
# =============================================================================

banner("STAGE26-8D4 :: PUSH")

try:
    from kaggle_secrets import UserSecretsClient
except Exception as exc:
    raise RuntimeError("Kaggle UserSecretsClient unavailable.") from exc

try:
    github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "Could not read Kaggle Secret GITHUB_TOKEN. Do not paste the token into notebook source."
    ) from exc

if not github_token or len(github_token.strip()) < 20:
    raise RuntimeError("GITHUB_TOKEN secret is empty or unusable.")

askpass_fd, askpass_name = tempfile.mkstemp(
    prefix="stage26_8d4_askpass_", suffix=".sh"
)
os.close(askpass_fd)
askpass_path = Path(askpass_name)

try:
    askpass_path.write_text(
        "#!/bin/sh\n"
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *Password*) printf "%s\\n" "$STAGE26_GITHUB_TOKEN" ;;\n'
        '  *) printf "%s\\n" "" ;;\n'
        "esac\n",
        encoding="utf-8",
    )
    askpass_path.chmod(
        askpass_path.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH
    )

    push_env = os.environ.copy()
    push_env["GIT_ASKPASS"] = str(askpass_path)
    push_env["GIT_TERMINAL_PROMPT"] = "0"
    push_env["STAGE26_GITHUB_TOKEN"] = github_token.strip()

    push_output = git("push", "origin", "main", env=push_env)
    print(push_output if push_output else "<push completed>")
finally:
    try:
        askpass_path.unlink(missing_ok=True)
    except Exception:
        pass
    github_token = None
    if "push_env" in locals():
        push_env.pop("STAGE26_GITHUB_TOKEN", None)


# =============================================================================
# 21. REMOTE BYTE VERIFICATION
# =============================================================================

banner("STAGE26-8D4 :: REMOTE BYTE VERIFICATION")

git("fetch", "origin", "main")
local_after = git("rev-parse", "HEAD")
remote_after = git("rev-parse", "origin/main")

print("Local HEAD :", local_after)
print("origin/main:", remote_after)

if local_after != new_head or remote_after != new_head:
    raise RuntimeError("Local/remote commit mismatch after Stage26-8D4 push.")

all_package_paths = [REPO / p for p in expected_paths]
verification = []

for path in all_package_paths:
    rel = str(path.relative_to(REPO))
    local_bytes = path.read_bytes()
    remote_bytes = git_blob_bytes("origin/main", rel)
    local_sha = sha256_bytes(local_bytes)
    remote_sha = sha256_bytes(remote_bytes)
    same = local_bytes == remote_bytes
    verification.append(
        {
            "path": rel,
            "local_sha256": local_sha,
            "remote_sha256": remote_sha,
            "byte_identical": same,
        }
    )
    print(f"{'PASS' if same else 'FAIL'} {rel}")
    if not same:
        raise RuntimeError(f"Remote byte verification failed: {rel}")


# =============================================================================
# 22. FINAL
# =============================================================================

final_status = git("status", "--porcelain")

banner("STAGE26-8D4 COMPLETE")

print("Durable figure-generation anchor:", new_head)
print("HEAD == origin/main           :", local_after == remote_after == new_head)
print("Repo clean                    :", final_status == "")
print(
    "Remote files byte-identical   :",
    all(item["byte_identical"] for item in verification),
)
print("Publication figures generated :", len(figure_records))
print("Rendered figure files         :", 16)

print("\nFIGURE IDS:")
for record in figure_records:
    print(" ", record["figure_id"])

print("\nSCIENTIFIC STATE:")
print("  quantitative inputs          : STAGE26-8D3 PUBLICATION TABLES ONLY")
print("  effective schema             : STAGE26-8D2 + STAGE26-8D2A")
print("  cold-start uncertainty       : NOT RENDERED")
print("  warm uncertainty             : STAGE26-6F1 CORRECTED ONLY")
print("  memory aggregation           : NONE")
print("  Stage26-7C                   : COPIED / NOT RECOMPUTED")
print("  Group-B ratio                : NOT RECOMPUTED / NOT INVERTED")
print("  Stage26-6D Pareto            : COPIED / NOT RECOMPUTED")
print("  historical Stage26-4C3       : RETAINED")
print("  Stage26-8 sensitivity        : DESCRIPTIVE / SEPARATE")
print("  complete E2E                 : UNAVAILABLE")
print("  timing                       : NO")
print("  inference                    : NO")
print("  bootstrap                    : NO")
print("  PCAP                         : NO")
print("  labels                       : NO")
print("  GPU                          : OFF")

if final_status:
    raise RuntimeError("Repository is not clean after Stage26-8D4.")

print("\nNEXT:")
print("  Run the CPU publication-artifact hash/protocol audit over")
print("  Stage26-8D3 tables + Stage26-8D4 figures before final CPU closure.")
print("  GPU remains OFF.")


STAGE26-8D4 :: DURABLE STATE GATE
Expected parent: b11936a4ee8234a36a29057658f7ee420a5cf68e
Local HEAD     : b11936a4ee8234a36a29057658f7ee420a5cf68e
origin/main    : b11936a4ee8234a36a29057658f7ee420a5cf68e
Repo clean     : True

STAGE26-8D4 :: EFFECTIVE FIGURE SCHEMA GATE
Expected effective schema SHA256: 973cc58ed6572d30b1cce94f3659226d99b185390c2aca92d9ed030e50358ad0
Actual effective schema SHA256  : 973cc58ed6572d30b1cce94f3659226d99b185390c2aca92d9ed030e50358ad0
Figure IDs:
  F26_WARM_LATENCY
  F26_WARM_THROUGHPUT
  F26_MEMORY_PACKAGE
  F26_COLD_START
  F26_CAPACITY_SCALING
  F26_PARETO
  F26_REPRESENTATION_SENSITIVITY
  F26_COMPONENT_BOUNDARY

STAGE26-8D4 :: STAGE26-8D3 TABLE IDENTITY GATES
T26_CPU_WARM_INFERENCE                                 expected=d664feca94308ea544506df7ef5c997bc4486f457153abfc1959db36808cfaad
                                                       actual  =d664feca94308ea544506df7ef5c997bc4486f457153abfc1959db36808cfaad
T26_CPU_MEMORY_PACKAGE            

In [21]:
# STAGE26-8D5 :: CPU PUBLICATION-ARTIFACT HASH / PROTOCOL AUDIT
from __future__ import annotations

import csv, hashlib, json, os, stat, subprocess, tempfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")
ROOT = REPO / "results" / "stage26_deployment_profiling"

EXPECTED_PARENT = "aefaa993e86dbc1d7fbd7e9c301e629883b0c51d"
TABLE_ANCHOR = "b11936a4ee8234a36a29057658f7ee420a5cf68e"
FIGURE_ANCHOR = EXPECTED_PARENT

PROTOCOL_SHA = "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
SCHEMA_SHA = "973cc58ed6572d30b1cce94f3659226d99b185390c2aca92d9ed030e50358ad0"
PUB_LABEL = "COMPONENT_LEVEL_WITH_STAGE26_8_4C3_SENSITIVITY_AUDIT_COMPLETED"

SCHEMA = ROOT / "stage26_8d2a_cpu_publication_schema_erratum" / "stage26_8d2a_effective_cpu_publication_schema.json"
TDIR = ROOT / "stage26_8d3_cpu_publication_tables"
FDIR = ROOT / "stage26_8d4_cpu_publication_figures"

TINDEX = TDIR / "stage26_8d3_cpu_publication_tables_index.json"
TRECEIPT = TDIR / "stage26_8d3_cpu_publication_tables_receipt.json"
TMANIFEST = TDIR / "stage26_8d3_cpu_publication_tables_manifest.json"

FINDEX = FDIR / "stage26_8d4_cpu_publication_figures_index.json"
FRECEIPT = FDIR / "stage26_8d4_cpu_publication_figures_receipt.json"
FMANIFEST = FDIR / "stage26_8d4_cpu_publication_figures_manifest.json"

EXPECTED_CONTROL_SHA = {
    TINDEX: "ae869df663d349a448f797789f285cdd6bfa6f7c98183ff30f4102a14ee374ea",
    TRECEIPT: "941eca833d7adaa48227097be858ca82e74364ba84ce3c42162a4568dd0e3a04",
    TMANIFEST: "656b0e936a18405e05d0775b1703b9417fd87afe52f4c610d72ef8df70971a97",
    FINDEX: "31807877ab1634f1bcd55fec69953e9820fbb0c6916ce6db1ab975f8841dd124",
    FRECEIPT: "5bc82460bf880133ae51de9bef78f783094ecae1f90cd97e72bead9a587adef1",
    FMANIFEST: "4ab72733b333c51086dd5ac0e643bcd21b2bb3f2edf268aefeebef79833f0a31",
}

TABLE_IDS = [
    "T26_CPU_WARM_INFERENCE",
    "T26_CPU_MEMORY_PACKAGE",
    "T26_CPU_COLD_START",
    "T26_CPU_CAPACITY_SCALING",
    "T26_COMPONENT_MEASUREMENTS",
    "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO",
    "T26_PARETO",
    "T26_REPRESENTATION_SENSITIVITY",
]
FIGURE_IDS = [
    "F26_WARM_LATENCY",
    "F26_WARM_THROUGHPUT",
    "F26_MEMORY_PACKAGE",
    "F26_COLD_START",
    "F26_CAPACITY_SCALING",
    "F26_PARETO",
    "F26_REPRESENTATION_SENSITIVITY",
    "F26_COMPONENT_BOUNDARY",
]
ROWS = {
    "T26_CPU_WARM_INFERENCE": 80,
    "T26_CPU_MEMORY_PACKAGE": 400,
    "T26_CPU_COLD_START": 56,
    "T26_CPU_CAPACITY_SCALING": 40,
    "T26_COMPONENT_MEASUREMENTS": 4,
    "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO": 7,
    "T26_PARETO": 6,
    "T26_REPRESENTATION_SENSITIVITY": 5,
}
BATCHES = [1, 64, 256, 1024, 8192]
CPU_MODES = {"CPU_1_PHYSICAL_CORE", "CPU_2_PHYSICAL_CORE"}
TARGETS = {
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
    "ENS_LGBM_XGB_EQUAL",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
}
PRIMARY = {
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
}
TPATH = {x: TDIR / f"{x}.csv" for x in TABLE_IDS}

OUT = ROOT / "stage26_8d5_cpu_publication_artifact_audit"
AUDIT = OUT / "stage26_8d5_cpu_publication_artifact_audit.json"
RECEIPT = OUT / "stage26_8d5_cpu_publication_artifact_audit_receipt.json"
MANIFEST = OUT / "stage26_8d5_cpu_publication_artifact_audit_manifest.json"
COMMIT_MSG = "stage26: audit CPU publication artifacts"

def banner(s):
    print("\n" + "=" * 124)
    print(s)
    print("=" * 124)

def run(cmd, *, env=None, check=True):
    p = subprocess.run(
        cmd,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        check=False,
    )
    if check and p.returncode:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(cmd)}\n{p.stdout}"
        )
    return p.stdout.strip()

def git(*args, env=None):
    return run(["git", *args], env=env)

def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def jread(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)

def cread(path):
    with Path(path).open("r", encoding="utf-8", newline="") as f:
        r = csv.DictReader(f)
        return r.fieldnames, list(r)

def blank(v):
    return str(v).strip() == ""

def iv(v, label):
    try:
        return int(str(v).strip())
    except Exception as e:
        raise RuntimeError(f"Invalid int {label}: {v!r}") from e

def fv(v, label):
    try:
        return float(str(v).strip())
    except Exception as e:
        raise RuntimeError(f"Invalid float {label}: {v!r}") from e

def atomic_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True, allow_nan=False)
        f.write("\n")
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def blob(ref, rel):
    p = subprocess.run(
        ["git", "show", f"{ref}:{rel}"],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )
    if p.returncode:
        raise RuntimeError(
            p.stderr.decode("utf-8", errors="replace")
        )
    return p.stdout

def check_manifest(m, label):
    out = []
    seen = set()

    for x in m["files"]:
        rel = x["path"]

        if rel in seen:
            raise RuntimeError(
                f"Duplicate {label} manifest path: {rel}"
            )

        seen.add(rel)
        p = REPO / rel

        if not p.is_file():
            raise FileNotFoundError(p)

        a = sha(p)
        z = p.stat().st_size

        if a != x["sha256"] or z != int(x["size_bytes"]):
            raise RuntimeError(
                f"{label} manifest mismatch: {rel}"
            )

        out.append(rel)

    return out


# =============================================================================
# DURABLE STATE
# =============================================================================

banner("STAGE26-8D5 :: DURABLE STATE GATE")

git("fetch", "origin", "main")

head = git("rev-parse", "HEAD")
remote = git("rev-parse", "origin/main")
status = git("status", "--porcelain")

print("Expected parent:", EXPECTED_PARENT)
print("Local HEAD     :", head)
print("origin/main    :", remote)
print("Repo clean     :", status == "")

if (
    head != EXPECTED_PARENT
    or remote != EXPECTED_PARENT
    or status
):
    raise RuntimeError(
        "Stage26-8D5 durable state gate failed."
    )

if OUT.exists():
    raise RuntimeError(
        f"8D5 output already exists: {OUT}"
    )


# =============================================================================
# SCHEMA / PROTOCOL
# =============================================================================

banner("STAGE26-8D5 :: SCHEMA / PROTOCOL GATE")

if sha(SCHEMA) != SCHEMA_SHA:
    raise RuntimeError(
        "Effective schema SHA mismatch."
    )

schema = jread(SCHEMA)

if schema["measurement_protocol_sha256"] != PROTOCOL_SHA:
    raise RuntimeError(
        "Protocol SHA mismatch."
    )

if schema["publication_label"] != PUB_LABEL:
    raise RuntimeError(
        "Publication label mismatch."
    )

if [x["id"] for x in schema["tables"]] != TABLE_IDS:
    raise RuntimeError(
        "Table schema IDs changed."
    )

if [x["id"] for x in schema["figures"]] != FIGURE_IDS:
    raise RuntimeError(
        "Figure schema IDs changed."
    )

scope = schema["scope"]

for k in [
    "gpu_allowed",
    "complete_e2e_measurement_available",
    "cross_group_pareto_allowed",
    "missing_cost_imputation_allowed",
]:
    if scope[k] is not False:
        raise RuntimeError(
            f"Unsafe schema scope: {k}"
        )

print("Protocol SHA :", PROTOCOL_SHA)
print("Schema SHA   :", SCHEMA_SHA)
print("Tables       : 8")
print("Figures      : 8")
print("GPU allowed  : False")


# =============================================================================
# CONTROL IDENTITIES
# =============================================================================

banner("STAGE26-8D5 :: CONTROL-FILE IDENTITY")

for p, expected in EXPECTED_CONTROL_SHA.items():
    actual = sha(p)

    print(
        f"{p.name}\n"
        f"  expected={expected}\n"
        f"  actual  ={actual}"
    )

    if actual != expected:
        raise RuntimeError(
            f"Control SHA mismatch: {p}"
        )

tm = jread(TMANIFEST)
fm = jread(FMANIFEST)

tm_paths = check_manifest(
    tm,
    "8D3",
)

fm_paths = check_manifest(
    fm,
    "8D4",
)

if len(tm_paths) != 10:
    raise RuntimeError(
        f"8D3 manifest expected 10 entries, got {len(tm_paths)}"
    )

if len(fm_paths) != 18:
    raise RuntimeError(
        f"8D4 manifest expected 18 entries, got {len(fm_paths)}"
    )

print("8D3 manifest entries: 10 PASS")
print("8D4 manifest entries: 18 PASS")


# =============================================================================
# HISTORICAL BYTE IDENTITY
# =============================================================================

banner("STAGE26-8D5 :: HISTORICAL BYTE IDENTITY")

table_pkg = [
    *(TPATH[x] for x in TABLE_IDS),
    TINDEX,
    TRECEIPT,
    TMANIFEST,
]

fig_pkg = [
    *(
        FDIR / f"{x}.{ext}"
        for x in FIGURE_IDS
        for ext in ("png", "pdf")
    ),
    FINDEX,
    FRECEIPT,
    FMANIFEST,
]

for label, anchor, paths in [
    ("8D3", TABLE_ANCHOR, table_pkg),
    ("8D4", FIGURE_ANCHOR, fig_pkg),
]:
    for p in paths:
        rel = str(
            p.relative_to(REPO)
        )

        ok = (
            p.read_bytes()
            ==
            blob(
                anchor,
                rel,
            )
        )

        print(
            f"{'PASS' if ok else 'FAIL'} "
            f"{label} {rel}"
        )

        if not ok:
            raise RuntimeError(
                f"{label} byte identity failed: {rel}"
            )


# =============================================================================
# LOAD TABLES
# =============================================================================

banner("STAGE26-8D5 :: LOAD TABLES")

H = {}
R = {}

for tid in TABLE_IDS:
    H[tid], R[tid] = cread(
        TPATH[tid]
    )

    print(
        f"{tid:52s} "
        f"rows={len(R[tid]):4d}"
    )

    if len(R[tid]) != ROWS[tid]:
        raise RuntimeError(
            f"{tid} row count changed."
        )


# =============================================================================
# 8D3 RECEIPT
# =============================================================================

tr = jread(
    TRECEIPT
)

if tr["effective_schema_sha256"] != SCHEMA_SHA:
    raise RuntimeError(
        "8D3 schema link mismatch."
    )

if tr["table_row_counts"] != ROWS:
    raise RuntimeError(
        "8D3 receipt row map mismatch."
    )

if tr["complete_e2e"] != "UNAVAILABLE":
    raise RuntimeError(
        "8D3 E2E policy mismatch."
    )

if (
    tr["historical_stage26_4c3"]
    !=
    "RETAINED_AS_ORIGINAL_MEASUREMENT"
):
    raise RuntimeError(
        "4C3 retention mismatch."
    )

if (
    tr["stage26_8_representation_sensitivity"]
    !=
    "DESCRIPTIVE_SEPARATE_NOT_REPLACEMENT"
):
    raise RuntimeError(
        "Sensitivity policy mismatch."
    )

for k, v in tr["scientific_safety"].items():
    if (
        isinstance(v, bool)
        and v is not False
    ):
        raise RuntimeError(
            f"8D3 scientific-safety flag unexpectedly true: {k}"
        )


# =============================================================================
# WARM TABLE
# =============================================================================

banner("STAGE26-8D5 :: WARM TABLE AUDIT")

warm = R[
    "T26_CPU_WARM_INFERENCE"
]

if H["T26_CPU_WARM_INFERENCE"] != [
    "target_id",
    "group",
    "cpu_mode",
    "batch_size",
    "status",
    "n_timed_observations",
    "p50_latency_ms",
    "p50_ci95_low_ms",
    "p50_ci95_high_ms",
    "p95_latency_ms",
    "p95_ci95_low_ms",
    "p95_ci95_high_ms",
    "p99_latency_ms",
    "p99_ci95_low_ms",
    "p99_ci95_high_ms",
    "median_throughput_samples_per_s",
    "throughput_ci95_low_samples_per_s",
    "throughput_ci95_high_samples_per_s",
    "resource_limit_outcome",
]:
    raise RuntimeError(
        "Warm header changed."
    )

sc = Counter(
    x["status"]
    for x in warm
)

if sc != Counter(
    {
        "PASS": 73,
        "RESOURCE_LIMIT_OOM": 2,
        "TIMEOUT_RESOURCE_LIMIT": 5,
    }
):
    raise RuntimeError(
        f"Warm status counts changed: {sc}"
    )

keys = set()

metrics = [
    x
    for x in H[
        "T26_CPU_WARM_INFERENCE"
    ]
    if (
        x.endswith("_ms")
        or "throughput" in x
    )
]

for x in warm:
    b = iv(
        x["batch_size"],
        "warm batch",
    )

    k = (
        x["target_id"],
        x["cpu_mode"],
        b,
    )

    if k in keys:
        raise RuntimeError(
            f"Duplicate warm key {k}"
        )

    keys.add(k)

    if (
        x["target_id"] not in TARGETS
        or x["cpu_mode"] not in CPU_MODES
        or b not in BATCHES
    ):
        raise RuntimeError(
            f"Unexpected warm geometry: {k}"
        )

    n = iv(
        x["n_timed_observations"],
        f"{k} n",
    )

    if x["status"] == "PASS":
        if (
            n <= 0
            or
            x["resource_limit_outcome"] != "NONE"
        ):
            raise RuntimeError(
                f"Bad PASS row {k}"
            )

        for f in [
            "p50_latency_ms",
            "p50_ci95_low_ms",
            "p50_ci95_high_ms",
            "p95_latency_ms",
            "p95_ci95_low_ms",
            "p95_ci95_high_ms",
            "median_throughput_samples_per_s",
            "throughput_ci95_low_samples_per_s",
            "throughput_ci95_high_samples_per_s",
        ]:
            fv(
                x[f],
                f"{k} {f}",
            )

        p99 = [
            "p99_latency_ms",
            "p99_ci95_low_ms",
            "p99_ci95_high_ms",
        ]

        if n >= 100:
            for f in p99:
                fv(
                    x[f],
                    f"{k} {f}",
                )
        else:
            if any(
                not blank(
                    x[f]
                )
                for f in p99
            ):
                raise RuntimeError(
                    f"p99 leaked for n<100 {k}"
                )

    else:
        if (
            n != 0
            or
            x["resource_limit_outcome"]
            !=
            x["status"]
        ):
            raise RuntimeError(
                f"Bad non-PASS row {k}"
            )

        if any(
            not blank(
                x[f]
            )
            for f in metrics
        ):
            raise RuntimeError(
                f"Non-PASS metric imputation detected {k}"
            )

if keys != {
    (t, c, b)
    for t in TARGETS
    for c in CPU_MODES
    for b in BATCHES
}:
    raise RuntimeError(
        "Warm 8x2x5 geometry changed."
    )

print("73 PASS / 2 OOM / 5 TIMEOUT: PASS")
print("Corrected-CI shape             : PASS")
print("Resource-limit imputation      : NONE")


# =============================================================================
# MEMORY TABLE
# =============================================================================

banner("STAGE26-8D5 :: MEMORY TABLE AUDIT")

mem = R[
    "T26_CPU_MEMORY_PACKAGE"
]

mset = {
    "median_baseline_rss_mib",
    "median_loaded_rss_mib",
    "median_peak_rss_mib",
    "median_delta_model_rss_mib",
    "median_delta_peak_rss_mib",
}

if {
    x["memory_metric_name"]
    for x in mem
} != mset:
    raise RuntimeError(
        "Memory metric set changed."
    )

cond = defaultdict(set)
pkg = defaultdict(set)
mkeys = set()

for x in mem:
    b = iv(
        x["batch_size"],
        "memory batch",
    )

    k = (
        x["target_id"],
        x["cpu_mode"],
        b,
        x["memory_metric_name"],
    )

    if k in mkeys:
        raise RuntimeError(
            f"Duplicate memory key {k}"
        )

    mkeys.add(k)

    c = k[:3]

    cond[c].add(
        k[3]
    )

    pkg[
        x["target_id"]
    ].add(
        x["deployment_package_size_mib"]
    )

    p = iv(
        x["planned_repetitions"],
        f"{c} planned",
    )

    q = iv(
        x["pass_repetitions"],
        f"{c} pass",
    )

    o = iv(
        x["resource_limit_oom_repetitions"],
        f"{c} oom",
    )

    to = iv(
        x["timeout_resource_limit_repetitions"],
        f"{c} timeout",
    )

    if (
        p != 5
        or
        q + o + to != p
    ):
        raise RuntimeError(
            f"Memory repetition accounting failed {c}"
        )

    if q > 0:
        fv(
            x["memory_value_mib"],
            f"{k} value",
        )
    elif not blank(
        x["memory_value_mib"]
    ):
        raise RuntimeError(
            f"Memory value imputed {k}"
        )

if (
    len(cond) != 80
    or
    any(
        v != mset
        for v in cond.values()
    )
):
    raise RuntimeError(
        "Memory condition geometry changed."
    )

if any(
    len(v) != 1
    for v in pkg.values()
):
    raise RuntimeError(
        "Package size varies by condition."
    )

print("400 rows = 80 conditions x 5 metrics: PASS")
print("CPU/batch aggregation                : NONE")
print("Unavailable-memory imputation        : NONE")


# =============================================================================
# OTHER TABLES
# =============================================================================

banner(
    "STAGE26-8D5 :: "
    "COLD / CAPACITY / COMPONENT / RATIO / PARETO / SENSITIVITY"
)

cold = R[
    "T26_CPU_COLD_START"
]

if any(
    "ci" in c.lower()
    for c in H[
        "T26_CPU_COLD_START"
    ]
):
    raise RuntimeError(
        "Cold CI column leaked."
    )

if {
    x["target_id"]
    for x in cold
} != TARGETS:
    raise RuntimeError(
        "Cold target universe changed."
    )

cc = Counter(
    x["target_id"]
    for x in cold
)

if set(
    cc.values()
) != {
    7
}:
    raise RuntimeError(
        f"Cold component count changed: {cc}"
    )

for x in cold:
    if (
        x["cpu_mode"]
        !=
        "CPU_1_PHYSICAL_CORE"
        or
        x["status"]
        !=
        "PASS"
        or
        iv(
            x["n_observations"],
            "cold n",
        )
        !=
        20
    ):
        raise RuntimeError(
            "Cold-start row invariant failed."
        )

    fv(
        x["p50_ms"],
        "cold p50",
    )

    fv(
        x["p95_ms"],
        "cold p95",
    )

print(
    "Cold-start: 56 point-estimate rows; CI ABSENT: PASS"
)


cap = R[
    "T26_CPU_CAPACITY_SCALING"
]

if any(
    "ci" in c.lower()
    for c in H[
        "T26_CPU_CAPACITY_SCALING"
    ]
):
    raise RuntimeError(
        "Capacity CI leaked."
    )

ck = {
    (
        x["target_id"],
        iv(
            x["batch_size"],
            "cap batch",
        ),
    )
    for x in cap
}

if ck != {
    (t, b)
    for t in TARGETS
    for b in BATCHES
}:
    raise RuntimeError(
        "Capacity geometry changed."
    )

print(
    "Capacity: 40 immutable target x batch rows; "
    "ratio CI ABSENT: PASS"
)


comp = R[
    "T26_COMPONENT_MEASUREMENTS"
]

if {
    x["component_family"]
    for x in comp
} != {
    "RAW_EXTRACTION_COMPONENT",
    "PACKET_IMAGE_REPRESENTATION_COMPONENT",
    "ISOLATED_WARM_INFERENCE_COMPONENT",
    "COMPLETE_E2E",
}:
    raise RuntimeError(
        "Component family set changed."
    )

e = [
    x
    for x in comp
    if x["component_family"]
    ==
    "COMPLETE_E2E"
]

if (
    len(e) != 1
    or
    e[0]["measurement_status"]
    !=
    "CLOSED_NO_VALID_COMPLETE_E2E_MEASUREMENT"
    or
    e[0]["complete_e2e_available"].lower()
    !=
    "false"
    or
    e[0]["quantitative_detail_reference"]
    !=
    "NONE"
):
    raise RuntimeError(
        "Complete-E2E closure row changed."
    )

print(
    "Component boundaries: "
    "3 measured components + E2E unavailable: PASS"
)


ratio = R[
    "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO"
]

if H[
    "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO"
] != [
    "target_id",
    "batch_size",
    "inference_over_representation_capacity_ratio",
    "publication_label",
]:
    raise RuntimeError(
        "Group-B ratio direction/header changed."
    )

if any(
    "ci" in c.lower()
    for c in H[
        "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO"
    ]
):
    raise RuntimeError(
        "Group-B ratio CI leaked."
    )

rk = {
    (
        x["target_id"],
        iv(
            x["batch_size"],
            "ratio batch",
        ),
    )
    for x in ratio
}

expected_rk = {
    ("STAGE20_MASKED_CNN_V1", 1),
    ("STAGE20_MASKED_CNN_V1", 64),
    ("STAGE20_MASKED_CNN_V1", 256),
    ("STAGE21_MASKED_VIT_V1", 1),
    ("STAGE21_MASKED_VIT_V1", 64),
    ("STAGE21_MASKED_VIT_V1", 256),
    ("STAGE21_MASKED_VIT_V1", 1024),
}

if rk != expected_rk:
    raise RuntimeError(
        "Group-B ratio matched-PASS geometry changed."
    )

for x in ratio:
    if (
        fv(
            x["inference_over_representation_capacity_ratio"],
            "ratio",
        )
        <= 0
        or
        x["publication_label"]
        !=
        PUB_LABEL
    ):
        raise RuntimeError(
            "Group-B ratio row invariant failed."
        )

print(
    "Group-B inference/representation ratio: "
    "7 rows; CI ABSENT: PASS"
)


par = R[
    "T26_PARETO"
]

if {
    x["target_id"]
    for x in par
} != PRIMARY:
    raise RuntimeError(
        "Pareto target universe changed."
    )

if Counter(
    x["group"]
    for x in par
) != Counter(
    {
        "GROUP_A_DUPSAFE70": 4,
        "GROUP_B_PACKET_IMAGE": 2,
    }
):
    raise RuntimeError(
        "Pareto group separation changed."
    )

pc = Counter(
    (
        x["group"],
        x["pareto_status"],
    )
    for x in par
)

if pc != Counter(
    {
        (
            "GROUP_A_DUPSAFE70",
            "FRONTIER_MEMBER",
        ): 3,
        (
            "GROUP_A_DUPSAFE70",
            "NOT_FRONTIER_MEMBER",
        ): 1,
        (
            "GROUP_B_PACKET_IMAGE",
            "FRONTIER_MEMBER",
        ): 1,
        (
            "GROUP_B_PACKET_IMAGE",
            "NOT_FRONTIER_MEMBER",
        ): 1,
    }
):
    raise RuntimeError(
        "Pareto membership changed."
    )

if any(
    "ci" in c.lower()
    for c in H[
        "T26_PARETO"
    ]
):
    raise RuntimeError(
        "Pareto CI leaked."
    )

print(
    "Pareto: within-group immutable membership; "
    "PR-AUC CI ABSENT: PASS"
)


sens = R[
    "T26_REPRESENTATION_SENSITIVITY"
]

if [
    iv(
        x["batch_size"],
        "sens batch",
    )
    for x in sens
] != BATCHES:
    raise RuntimeError(
        "Sensitivity batches changed."
    )

for x in sens:
    b = iv(
        x["batch_size"],
        "sens batch",
    )

    if (
        iv(
            x["n_pairs"],
            f"B{b} pairs",
        )
        !=
        5
        or
        iv(
            x["fingerprints_equal_count"],
            f"B{b} fp",
        )
        !=
        5
    ):
        raise RuntimeError(
            f"Sensitivity geometry failed B{b}"
        )

    if fv(
        x["median_throughput_ratio_v3_over_v2"],
        f"B{b} ratio",
    ) <= 0:
        raise RuntimeError(
            f"Sensitivity ratio invalid B{b}"
        )

print(
    "Sensitivity: "
    "5 batches / 25 pairs / 25 fingerprint matches: PASS"
)


# =============================================================================
# FIGURE PACKAGE
# =============================================================================

banner(
    "STAGE26-8D5 :: "
    "FIGURE INDEX / RECEIPT / FILE STRUCTURE"
)

fi = jread(
    FINDEX
)

fr = jread(
    FRECEIPT
)

if (
    fi["effective_schema_sha256"]
    !=
    SCHEMA_SHA
    or
    fi["publication_label"]
    !=
    PUB_LABEL
):
    raise RuntimeError(
        "Figure index schema/publication link failed."
    )

if (
    fi["quantitative_input_policy"]
    !=
    "STAGE26_8D3_PUBLICATION_TABLES_ONLY"
):
    raise RuntimeError(
        "Figure quantitative input policy changed."
    )

if [
    x["figure_id"]
    for x in fi["figures"]
] != FIGURE_IDS:
    raise RuntimeError(
        "Figure index IDs changed."
    )

if (
    fi["logical_figure_count"] != 8
    or
    fi["rendered_file_count"] != 16
):
    raise RuntimeError(
        "Figure count changed."
    )

required_tokens = {
    "F26_WARM_LATENCY": {
        "CORRECTED_STAGE26_6F1_UNCERTAINTY_ONLY",
        "NO_STAGE26_2_HISTORICAL_CI",
        "RESOURCE_LIMITS_EXPLICIT",
        "NO_INTERPOLATION_THROUGH_RESOURCE_LIMITS",
    },
    "F26_WARM_THROUGHPUT": {
        "CORRECTED_STAGE26_6F1_UNCERTAINTY_ONLY",
        "RESOURCE_LIMITS_EXPLICIT",
        "NO_POST_HOC_BEST_BATCH",
    },
    "F26_MEMORY_PACKAGE": {
        "PACKAGE_SIZE_INDEPENDENT_PANEL",
        "ALL_FIVE_FROZEN_MEMORY_METRICS",
        "NO_CPU_OR_BATCH_AGGREGATION",
        "RAM_LIMITS_EXPLICIT",
        "NO_IMPUTATION",
    },
    "F26_COLD_START": {
        "POINT_ESTIMATES_ONLY",
        "NO_COLD_START_CI",
    },
    "F26_CAPACITY_SCALING": {
        "STAGE26_7C_COPIED_NOT_RECOMPUTED",
        "NO_RATIO_CI",
        "NO_POST_HOC_BEST_BATCH",
        "COMPONENT_LEVEL_ONLY",
    },
    "F26_PARETO": {
        "GROUPS_SEPARATE",
        "STAGE26_6D_MEMBERSHIP_COPIED",
        "NO_CROSS_GROUP_FRONTIER",
        "NO_PR_AUC_BOOTSTRAP",
    },
    "F26_REPRESENTATION_SENSITIVITY": {
        "DESCRIPTIVE_ONLY",
        "NEUTRAL_RATIO_ONE_REFERENCE",
        "NO_SIGNIFICANCE_TEST",
        "NO_FIXED_BIAS_CLAIM",
        "V3_NOT_REPLACEMENT",
    },
    "F26_COMPONENT_BOUNDARY": {
        "DISTINCT_COMPONENTS",
        "COMPLETE_E2E_UNAVAILABLE",
        "NO_ADDITIVE_LATENCY",
        "NO_SYNTHETIC_PIPELINE_THROUGHPUT",
        "NO_MISSING_COST_IMPUTATION",
    },
}

idxfiles = {}

for x in fi["figures"]:
    if not required_tokens[
        x["figure_id"]
    ].issubset(
        set(
            x["validations"]
        )
    ):
        raise RuntimeError(
            f"Figure validation tokens missing: "
            f"{x['figure_id']}"
        )

    if (
        {
            f["format"]
            for f in x["files"]
        }
        !=
        {
            "PNG",
            "PDF",
        }
        or
        len(
            x["files"]
        )
        !=
        2
    ):
        raise RuntimeError(
            f"Figure formats changed: "
            f"{x['figure_id']}"
        )

    for f in x["files"]:
        idxfiles[
            f["path"]
        ] = f

for k in [
    "bootstrap_executed",
    "cold_start_ci_rendered",
    "complete_e2e_measurement_available",
    "gpu_used",
    "inference_executed",
    "labels_accessed",
    "memory_condition_aggregation",
    "model_loaded",
    "pareto_recomputed",
    "pcap_accessed",
    "random_sampling_executed",
    "ratio_inverted",
    "ratio_recomputed",
    "raw_measurement_files_read",
    "stage26_2_historical_ci_rendered",
    "stage26_7c_recomputed",
    "timing_executed",
]:
    if fr[k] is not False:
        raise RuntimeError(
            f"8D4 unsafe receipt flag: {k}"
        )

if (
    fr["effective_schema_sha256"]
    !=
    SCHEMA_SHA
    or
    fr["quantitative_inputs"]
    !=
    "STAGE26_8D3_PUBLICATION_TABLES_ONLY"
):
    raise RuntimeError(
        "8D4 receipt linkage failed."
    )

for tid, expected in fr[
    "table_sha256"
].items():
    if sha(
        TPATH[tid]
    ) != expected:
        raise RuntimeError(
            f"8D4 input table hash drifted: {tid}"
        )

mf = {
    x["path"]: x
    for x in fm["files"]
    if x["path"].lower().endswith(
        (
            ".png",
            ".pdf",
        )
    )
}

if set(mf) != set(idxfiles):
    raise RuntimeError(
        "Figure index/manifest file set mismatch."
    )

for rel, x in idxfiles.items():
    p = REPO / rel

    if (
        x["sha256"]
        !=
        mf[rel]["sha256"]
        or
        int(
            x["size_bytes"]
        )
        !=
        int(
            mf[rel]["size_bytes"]
        )
    ):
        raise RuntimeError(
            f"Figure index/manifest metadata mismatch: {rel}"
        )

    prefix = p.read_bytes()[:8]

    if (
        rel.endswith(".png")
        and
        prefix
        !=
        b"\x89PNG\r\n\x1a\n"
    ):
        raise RuntimeError(
            f"Bad PNG signature: {rel}"
        )

    if (
        rel.endswith(".pdf")
        and
        not prefix.startswith(
            b"%PDF-"
        )
    ):
        raise RuntimeError(
            f"Bad PDF signature: {rel}"
        )

    if p.stat().st_size < 1000:
        raise RuntimeError(
            f"Rendered figure too small: {rel}"
        )

print("8 figure families / 16 rendered files: PASS")
print("Required validation tokens            : PASS")
print("PNG/PDF signatures                    : PASS")
print("8D3-table-only quantitative input     : PASS")


# =============================================================================
# CROSS-PACKAGE VERDICT
# =============================================================================

banner(
    "STAGE26-8D5 :: CROSS-PACKAGE PROTOCOL VERDICT"
)

rules = schema[
    "global_rules"
]

if (
    "Point estimates only"
    not in
    rules[
        "cold_start_uncertainty"
    ]
):
    raise RuntimeError(
        "Cold rule missing."
    )

if (
    "Stage26-6F1"
    not in
    rules[
        "warm_cpu_uncertainty"
    ]
):
    raise RuntimeError(
        "Warm CI rule missing."
    )

if (
    "unavailable"
    not in
    rules[
        "complete_e2e"
    ].lower()
):
    raise RuntimeError(
        "E2E rule missing."
    )

if (
    "Never recompute"
    not in
    rules[
        "pareto"
    ]
):
    raise RuntimeError(
        "Pareto rule missing."
    )

if (
    "Retain historical Stage26-4C3"
    not in
    rules[
        "representation"
    ]
):
    raise RuntimeError(
        "4C3 rule missing."
    )

print("Cold-start uncertainty : POINT ESTIMATES ONLY")
print("Warm uncertainty       : STAGE26-6F1 CORRECTED ONLY")
print("Memory aggregation     : NONE")
print("Stage26-7C             : UNCHANGED")
print("Ratio recomputation    : NONE")
print("Ratio CI               : NONE")
print("Pareto recomputation   : NONE")
print("Cross-group Pareto     : NONE")
print("PR-AUC bootstrap CI    : NONE")
print("Historical Stage26-4C3 : RETAINED")
print("Stage26-8 sensitivity  : DESCRIPTIVE / SEPARATE")
print("Complete E2E           : UNAVAILABLE")
print("GPU                    : OFF")


# =============================================================================
# WRITE DURABLE AUDIT
# =============================================================================

banner(
    "STAGE26-8D5 :: WRITE DURABLE AUDIT"
)

OUT.mkdir(
    parents=True,
    exist_ok=False,
)

now = datetime.now(
    timezone.utc
).isoformat()

audit = {
    "schema":
        "stage26_8d5_cpu_publication_artifact_audit_v1",

    "created_at_utc":
        now,

    "scientific_parent":
        EXPECTED_PARENT,

    "measurement_protocol_sha256":
        PROTOCOL_SHA,

    "effective_schema_sha256":
        SCHEMA_SHA,

    "stage26_8d3_anchor":
        TABLE_ANCHOR,

    "stage26_8d4_anchor":
        FIGURE_ANCHOR,

    "table_count":
        8,

    "logical_figure_count":
        8,

    "rendered_figure_file_count":
        16,

    "table_row_counts":
        ROWS,

    "warm_status_counts": {
        "PASS":
            73,

        "RESOURCE_LIMIT_OOM":
            2,

        "TIMEOUT_RESOURCE_LIMIT":
            5,
    },

    "scientific_state": {
        "cold_start_publication":
            "POINT_ESTIMATES_ONLY",

        "warm_uncertainty":
            "STAGE26_6F1_CORRECTED_ONLY",

        "memory_aggregation":
            False,

        "stage26_7c_recomputed":
            False,

        "ratio_recomputed":
            False,

        "ratio_inverted":
            False,

        "ratio_ci":
            False,

        "pareto_recomputed":
            False,

        "cross_group_pareto":
            False,

        "pr_auc_bootstrap_ci":
            False,

        "historical_stage26_4c3":
            "RETAINED",

        "stage26_8_sensitivity":
            "DESCRIPTIVE_SEPARATE_NOT_REPLACEMENT",

        "complete_e2e":
            "UNAVAILABLE",
    },

    "non_computation": {
        "timing":
            False,

        "inference":
            False,

        "model_loading":
            False,

        "bootstrap":
            False,

        "random_sampling":
            False,

        "pcap":
            False,

        "labels":
            False,

        "gpu":
            False,
    },

    "verdict":
        "CPU_PUBLICATION_ARTIFACT_PACKAGE_PROTOCOL_CONFORMANT",

    "gpu_allowed_after_this_stage":
        False,

    "next_stage":
        (
            "STAGE26_8D6_FINAL_CPU_CLOSURE_AND_"
            "FRESH_SESSION_GPU_BOOTSTRAP_MANIFEST"
        ),
}

atomic_json(
    AUDIT,
    audit,
)

audit_sha = sha(
    AUDIT
)

receipt = {
    "schema":
        "stage26_8d5_cpu_publication_artifact_audit_receipt_v1",

    "created_at_utc":
        now,

    "scientific_parent":
        EXPECTED_PARENT,

    "measurement_protocol_sha256":
        PROTOCOL_SHA,

    "effective_schema_sha256":
        SCHEMA_SHA,

    "stage26_8d3_manifest_sha256":
        EXPECTED_CONTROL_SHA[
            TMANIFEST
        ],

    "stage26_8d4_manifest_sha256":
        EXPECTED_CONTROL_SHA[
            FMANIFEST
        ],

    "audit_sha256":
        audit_sha,

    "verdict":
        "CPU_PUBLICATION_ARTIFACT_PACKAGE_PROTOCOL_CONFORMANT",

    "gpu_allowed":
        False,
}

atomic_json(
    RECEIPT,
    receipt,
)

receipt_sha = sha(
    RECEIPT
)

manifest = {
    "schema":
        "stage26_8d5_cpu_publication_artifact_audit_manifest_v1",

    "created_at_utc":
        now,

    "scientific_parent":
        EXPECTED_PARENT,

    "files": [
        {
            "path":
                str(
                    AUDIT.relative_to(
                        REPO
                    )
                ),

            "sha256":
                audit_sha,

            "size_bytes":
                AUDIT.stat().st_size,
        },
        {
            "path":
                str(
                    RECEIPT.relative_to(
                        REPO
                    )
                ),

            "sha256":
                receipt_sha,

            "size_bytes":
                RECEIPT.stat().st_size,
        },
    ],
}

atomic_json(
    MANIFEST,
    manifest,
)

manifest_sha = sha(
    MANIFEST
)

print(
    "Audit SHA256   :",
    audit_sha,
)

print(
    "Receipt SHA256 :",
    receipt_sha,
)

print(
    "Manifest SHA256:",
    manifest_sha,
)


# =============================================================================
# PRE-COMMIT
# =============================================================================

banner(
    "STAGE26-8D5 :: PRE-COMMIT GIT AUDIT"
)

lines = [
    x
    for x in git(
        "status",
        "--porcelain",
    ).splitlines()
    if x.strip()
]

for x in lines:
    print(
        x
    )

prefix = (
    "?? "
    +
    str(
        OUT.relative_to(
            REPO
        )
    )
)

if (
    not lines
    or
    any(
        not x.startswith(
            prefix
        )
        for x in lines
    )
):
    raise RuntimeError(
        "Unexpected pre-commit repository state."
    )


# =============================================================================
# COMMIT
# =============================================================================

banner(
    "STAGE26-8D5 :: COMMIT"
)

git(
    "add",
    str(
        OUT.relative_to(
            REPO
        )
    ),
)

staged = git(
    "diff",
    "--cached",
    "--name-only",
).splitlines()

expected = sorted(
    str(
        x.relative_to(
            REPO
        )
    )
    for x in [
        AUDIT,
        RECEIPT,
        MANIFEST,
    ]
)

for x in staged:
    print(
        " ",
        x
    )

if sorted(
    staged
) != expected:
    raise RuntimeError(
        "Unexpected staged file set."
    )

git(
    "commit",
    "-m",
    COMMIT_MSG,
)

new_head = git(
    "rev-parse",
    "HEAD",
)

parent = git(
    "rev-parse",
    "HEAD^",
)

subject = git(
    "log",
    "-1",
    "--pretty=%s",
)

print(
    "Parent :",
    parent,
)

print(
    "HEAD   :",
    new_head,
)

print(
    "Subject:",
    subject,
)

if (
    parent != EXPECTED_PARENT
    or
    subject != COMMIT_MSG
):
    raise RuntimeError(
        "8D5 commit identity failed."
    )


# =============================================================================
# PUSH
# =============================================================================

banner(
    "STAGE26-8D5 :: PUSH"
)

from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)

if (
    not token
    or
    len(
        token.strip()
    )
    <
    20
):
    raise RuntimeError(
        "GITHUB_TOKEN unavailable."
    )

fd, name = tempfile.mkstemp(
    prefix="stage26_8d5_askpass_",
    suffix=".sh",
)

os.close(
    fd
)

ask = Path(
    name
)

try:
    ask.write_text(
        "#!/bin/sh\n"
        "case \"$1\" in\n"
        "  *Username*) printf \"%s\\n\" \"x-access-token\" ;;\n"
        "  *Password*) printf \"%s\\n\" \"$STAGE26_GITHUB_TOKEN\" ;;\n"
        "  *) printf \"%s\\n\" \"\" ;;\n"
        "esac\n",
        encoding="utf-8",
    )

    ask.chmod(
        ask.stat().st_mode
        |
        stat.S_IXUSR
        |
        stat.S_IXGRP
        |
        stat.S_IXOTH
    )

    env = os.environ.copy()

    env[
        "GIT_ASKPASS"
    ] = str(
        ask
    )

    env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    env[
        "STAGE26_GITHUB_TOKEN"
    ] = token.strip()

    print(
        git(
            "push",
            "origin",
            "main",
            env=env,
        )
    )

finally:
    ask.unlink(
        missing_ok=True
    )

    token = None

    if "env" in locals():
        env.pop(
            "STAGE26_GITHUB_TOKEN",
            None,
        )


# =============================================================================
# REMOTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-8D5 :: REMOTE BYTE VERIFICATION"
)

git(
    "fetch",
    "origin",
    "main",
)

lh = git(
    "rev-parse",
    "HEAD",
)

rh = git(
    "rev-parse",
    "origin/main",
)

print(
    "Local HEAD :",
    lh,
)

print(
    "origin/main:",
    rh,
)

if (
    lh != new_head
    or
    rh != new_head
):
    raise RuntimeError(
        "Local/remote commit mismatch."
    )

for p in [
    AUDIT,
    RECEIPT,
    MANIFEST,
]:
    rel = str(
        p.relative_to(
            REPO
        )
    )

    lb = p.read_bytes()

    rb = blob(
        "origin/main",
        rel,
    )

    ok = (
        lb == rb
    )

    print(
        f"{'PASS' if ok else 'FAIL'} "
        f"{rel}"
    )

    print(
        "  local :",
        hashlib.sha256(
            lb
        ).hexdigest(),
    )

    print(
        "  remote:",
        hashlib.sha256(
            rb
        ).hexdigest(),
    )

    if not ok:
        raise RuntimeError(
            f"Remote byte mismatch: {rel}"
        )


# =============================================================================
# COMPLETE
# =============================================================================

final = git(
    "status",
    "--porcelain",
)

banner(
    "STAGE26-8D5 COMPLETE"
)

print(
    "Durable audit anchor        :",
    new_head,
)

print(
    "HEAD == origin/main         :",
    lh == rh == new_head,
)

print(
    "Repo clean                  :",
    final == "",
)

print(
    "Publication tables          : 8 / PASS"
)

print(
    "Publication figure families : 8 / PASS"
)

print(
    "Rendered figure files       : 16 / PASS"
)

print(
    "Protocol verdict            : "
    "CPU_PUBLICATION_ARTIFACT_PACKAGE_PROTOCOL_CONFORMANT"
)

print(
    "GPU                          : OFF"
)

print(
    "GPU allowed yet              : NO"
)

print(
    "\nNEXT:"
)

print(
    "  Stage26-8D6 final CPU closure + "
    "fresh-session GPU bootstrap manifest."
)

print(
    "  Freeze exact repo/corpus/model/artifact recovery "
    "identities for the expected Kaggle reset."
)

if final:
    raise RuntimeError(
        "Repository not clean after Stage26-8D5."
    )


STAGE26-8D5 :: DURABLE STATE GATE
Expected parent: aefaa993e86dbc1d7fbd7e9c301e629883b0c51d
Local HEAD     : aefaa993e86dbc1d7fbd7e9c301e629883b0c51d
origin/main    : aefaa993e86dbc1d7fbd7e9c301e629883b0c51d
Repo clean     : True

STAGE26-8D5 :: SCHEMA / PROTOCOL GATE
Protocol SHA : d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
Schema SHA   : 973cc58ed6572d30b1cce94f3659226d99b185390c2aca92d9ed030e50358ad0
Tables       : 8
Figures      : 8
GPU allowed  : False

STAGE26-8D5 :: CONTROL-FILE IDENTITY
stage26_8d3_cpu_publication_tables_index.json
  expected=ae869df663d349a448f797789f285cdd6bfa6f7c98183ff30f4102a14ee374ea
  actual  =ae869df663d349a448f797789f285cdd6bfa6f7c98183ff30f4102a14ee374ea
stage26_8d3_cpu_publication_tables_receipt.json
  expected=941eca833d7adaa48227097be858ca82e74364ba84ce3c42162a4568dd0e3a04
  actual  =941eca833d7adaa48227097be858ca82e74364ba84ce3c42162a4568dd0e3a04
stage26_8d3_cpu_publication_tables_manifest.json
  expected=656b0e936a18405e05d

RuntimeError: Capacity CI leaked.

In [22]:
# =============================================================================
# STAGE26-8D5R1
# CAPACITY "CI" FALSE-POSITIVE DIAGNOSTIC
#
# READ ONLY
# =============================================================================

from __future__ import annotations

import csv
import subprocess
from pathlib import Path


REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CAPACITY = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_8d3_cpu_publication_tables"
    / "T26_CPU_CAPACITY_SCALING.csv"
)

EXPECTED_HEAD = (
    "aefaa993e86dbc1d7fbd7e9c301e629883b0c51d"
)

AUDIT_OUT = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_8d5_cpu_publication_artifact_audit"
)


def banner(text):
    print("\n" + "=" * 118)
    print(text)
    print("=" * 118)


def git(*args):
    p = subprocess.run(
        ["git", *args],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(args)} failed:\n{p.stdout}"
        )

    return p.stdout.strip()


# =============================================================================
# 1. DURABLE STATE
# =============================================================================

banner(
    "STAGE26-8D5R1 :: DURABLE STATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)

print(
    "Expected HEAD :",
    EXPECTED_HEAD,
)

print(
    "Local HEAD    :",
    head,
)

print(
    "origin/main   :",
    remote,
)

print(
    "Repo clean    :",
    status == "",
)

print(
    "8D5 output exists:",
    AUDIT_OUT.exists(),
)


if head != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected local HEAD."
    )


if remote != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected origin/main."
    )


if status:
    raise RuntimeError(
        "Repository is not clean."
    )


if AUDIT_OUT.exists():
    raise RuntimeError(
        "Partial Stage26-8D5 output exists unexpectedly."
    )


# =============================================================================
# 2. READ CAPACITY HEADER
# =============================================================================

banner(
    "STAGE26-8D5R1 :: CAPACITY HEADER"
)

with CAPACITY.open(
    "r",
    encoding="utf-8",
    newline="",
) as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    rows = list(reader)


print(
    "Rows:",
    len(rows),
)

print(
    "Columns:"
)

for col in header:
    print(
        " ",
        col
    )


if len(rows) != 40:
    raise RuntimeError(
        "Expected 40 capacity rows."
    )


expected_header = [
    "target_id",
    "group",
    "batch_size",
    "cpu1_status",
    "cpu2_status",
    "cpu1_throughput_multiplier_vs_b1",
    "cpu2_throughput_multiplier_vs_b1",
    "cpu2_over_cpu1_speedup",
    "physical_core_parallel_efficiency",
]


if header != expected_header:
    raise RuntimeError(
        "Capacity header differs from frozen Stage26-8D3 schema."
    )


# =============================================================================
# 3. REPRODUCE THE BAD AUDIT TEST
# =============================================================================

banner(
    "STAGE26-8D5R1 :: REPRODUCE FALSE POSITIVE"
)

substring_hits = [
    col
    for col in header
    if "ci" in col.lower()
]


print(
    'Columns matching naive `"ci" in name`:'
)

for col in substring_hits:
    print(
        " ",
        col
    )


if substring_hits != [
    "physical_core_parallel_efficiency"
]:
    raise RuntimeError(
        "Unexpected naive CI substring-hit set."
    )


print(
    "\nConfirmed:"
)

print(
    "  'efficiency' contains the character sequence 'ci'."
)

print(
    "  This is NOT a confidence-interval column."
)


# =============================================================================
# 4. TOKEN-AWARE CI DETECTION
# =============================================================================

banner(
    "STAGE26-8D5R1 :: TOKEN-AWARE CI DETECTION"
)

def is_ci_column(name: str) -> bool:
    n = name.lower()

    patterns = (
        "_ci_",
        "_ci95_",
        "_ci90_",
        "_ci99_",
        "ci_low",
        "ci_high",
        "ci95_low",
        "ci95_high",
        "confidence_interval",
        "confidence_low",
        "confidence_high",
    )

    if n == "ci":
        return True

    if n.startswith(
        "ci_"
    ):
        return True

    if n.endswith(
        "_ci"
    ):
        return True

    return any(
        p in n
        for p in patterns
    )


real_ci_hits = [
    col
    for col in header
    if is_ci_column(
        col
    )
]


print(
    "Real CI-style capacity columns:",
    real_ci_hits,
)


if real_ci_hits:
    raise RuntimeError(
        "Actual confidence-interval column detected in capacity table."
    )


# =============================================================================
# 5. CHECK OTHER CAPACITY INVARIANTS
# =============================================================================

banner(
    "STAGE26-8D5R1 :: CAPACITY STRUCTURAL CHECK"
)

targets = {
    row[
        "target_id"
    ]
    for row in rows
}

batches = {
    int(
        row[
            "batch_size"
        ]
    )
    for row in rows
}


print(
    "Target count:",
    len(targets),
)

print(
    "Batches     :",
    sorted(
        batches
    ),
)


if len(
    targets
) != 8:
    raise RuntimeError(
        "Expected 8 capacity targets."
    )


if sorted(
    batches
) != [
    1,
    64,
    256,
    1024,
    8192,
]:
    raise RuntimeError(
        "Capacity batch geometry changed."
    )


keys = {
    (
        row[
            "target_id"
        ],
        int(
            row[
                "batch_size"
            ]
        ),
    )
    for row in rows
}


if len(
    keys
) != 40:
    raise RuntimeError(
        "Expected 40 unique target × batch capacity rows."
    )


print(
    "Target x batch geometry: PASS"
)

print(
    "Ratio CI columns       : ABSENT"
)

print(
    "Parallel efficiency    : LEGITIMATE STAGE26-7C FIELD"
)


# =============================================================================
# 6. FINAL VERDICT
# =============================================================================

banner(
    "STAGE26-8D5R1 COMPLETE"
)

print(
    "Scientific result changed       : NO"
)

print(
    "Stage26-7C changed              : NO"
)

print(
    "Capacity ratios recomputed      : NO"
)

print(
    "Confidence intervals discovered : NO"
)

print(
    "Repository modified             : NO"
)

print(
    "Publication artifact written    : NO"
)

print(
    "GPU                             : OFF"
)

print(
    "\nROOT CAUSE:"
)

print(
    "  Stage26-8D5 used substring matching for 'ci'."
)

print(
    "  physical_core_parallel_efficiency contains 'ci' inside"
)

print(
    "  the ordinary English word 'efficiency'."
)

print(
    "\nRECOVERY_DECISION:"
)

print(
    "  PATCH_STAGE26_8D5_TOKEN_AWARE_CI_COLUMN_DETECTION"
)


STAGE26-8D5R1 :: DURABLE STATE
Expected HEAD : aefaa993e86dbc1d7fbd7e9c301e629883b0c51d
Local HEAD    : aefaa993e86dbc1d7fbd7e9c301e629883b0c51d
origin/main   : aefaa993e86dbc1d7fbd7e9c301e629883b0c51d
Repo clean    : True
8D5 output exists: False

STAGE26-8D5R1 :: CAPACITY HEADER
Rows: 40
Columns:
  target_id
  group
  batch_size
  cpu1_status
  cpu2_status
  cpu1_throughput_multiplier_vs_b1
  cpu2_throughput_multiplier_vs_b1
  cpu2_over_cpu1_speedup
  physical_core_parallel_efficiency

STAGE26-8D5R1 :: REPRODUCE FALSE POSITIVE
Columns matching naive `"ci" in name`:
  physical_core_parallel_efficiency

Confirmed:
  'efficiency' contains the character sequence 'ci'.
  This is NOT a confidence-interval column.

STAGE26-8D5R1 :: TOKEN-AWARE CI DETECTION
Real CI-style capacity columns: []

STAGE26-8D5R1 :: CAPACITY STRUCTURAL CHECK
Target count: 8
Batches     : [1, 64, 256, 1024, 8192]
Target x batch geometry: PASS
Ratio CI columns       : ABSENT
Parallel efficiency    : LEGITIMATE STAGE

In [23]:
# =============================================================================
# STAGE26-8D5 :: CPU PUBLICATION-ARTIFACT HASH / PROTOCOL AUDIT
# CORRECTED AFTER 8D5R1:
#   token-aware CI-column detection replaces naive substring matching.
#
# NO SCIENCE IS RE-RUN.
# GPU REMAINS OFF.
# =============================================================================

from __future__ import annotations

import csv
import hashlib
import json
import os
import stat
import subprocess
import tempfile

from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path


# =============================================================================
# CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

ROOT = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
)

EXPECTED_PARENT = (
    "aefaa993e86dbc1d7fbd7e9c301e629883b0c51d"
)

TABLE_ANCHOR = (
    "b11936a4ee8234a36a29057658f7ee420a5cf68e"
)

FIGURE_ANCHOR = EXPECTED_PARENT

PROTOCOL_SHA = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

SCHEMA_SHA = (
    "973cc58ed6572d30b1cce94f3659226d99b185390c2aca92d9ed030e50358ad0"
)

PUB_LABEL = (
    "COMPONENT_LEVEL_WITH_STAGE26_8_4C3_SENSITIVITY_AUDIT_COMPLETED"
)

SCHEMA = (
    ROOT
    / "stage26_8d2a_cpu_publication_schema_erratum"
    / "stage26_8d2a_effective_cpu_publication_schema.json"
)

TDIR = (
    ROOT
    / "stage26_8d3_cpu_publication_tables"
)

FDIR = (
    ROOT
    / "stage26_8d4_cpu_publication_figures"
)

TINDEX = (
    TDIR
    / "stage26_8d3_cpu_publication_tables_index.json"
)

TRECEIPT = (
    TDIR
    / "stage26_8d3_cpu_publication_tables_receipt.json"
)

TMANIFEST = (
    TDIR
    / "stage26_8d3_cpu_publication_tables_manifest.json"
)

FINDEX = (
    FDIR
    / "stage26_8d4_cpu_publication_figures_index.json"
)

FRECEIPT = (
    FDIR
    / "stage26_8d4_cpu_publication_figures_receipt.json"
)

FMANIFEST = (
    FDIR
    / "stage26_8d4_cpu_publication_figures_manifest.json"
)


EXPECTED_CONTROL_SHA = {
    TINDEX:
        "ae869df663d349a448f797789f285cdd6bfa6f7c98183ff30f4102a14ee374ea",

    TRECEIPT:
        "941eca833d7adaa48227097be858ca82e74364ba84ce3c42162a4568dd0e3a04",

    TMANIFEST:
        "656b0e936a18405e05d0775b1703b9417fd87afe52f4c610d72ef8df70971a97",

    FINDEX:
        "31807877ab1634f1bcd55fec69953e9820fbb0c6916ce6db1ab975f8841dd124",

    FRECEIPT:
        "5bc82460bf880133ae51de9bef78f783094ecae1f90cd97e72bead9a587adef1",

    FMANIFEST:
        "4ab72733b333c51086dd5ac0e643bcd21b2bb3f2edf268aefeebef79833f0a31",
}


TABLE_IDS = [
    "T26_CPU_WARM_INFERENCE",
    "T26_CPU_MEMORY_PACKAGE",
    "T26_CPU_COLD_START",
    "T26_CPU_CAPACITY_SCALING",
    "T26_COMPONENT_MEASUREMENTS",
    "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO",
    "T26_PARETO",
    "T26_REPRESENTATION_SENSITIVITY",
]


FIGURE_IDS = [
    "F26_WARM_LATENCY",
    "F26_WARM_THROUGHPUT",
    "F26_MEMORY_PACKAGE",
    "F26_COLD_START",
    "F26_CAPACITY_SCALING",
    "F26_PARETO",
    "F26_REPRESENTATION_SENSITIVITY",
    "F26_COMPONENT_BOUNDARY",
]


ROWS = {
    "T26_CPU_WARM_INFERENCE": 80,
    "T26_CPU_MEMORY_PACKAGE": 400,
    "T26_CPU_COLD_START": 56,
    "T26_CPU_CAPACITY_SCALING": 40,
    "T26_COMPONENT_MEASUREMENTS": 4,
    "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO": 7,
    "T26_PARETO": 6,
    "T26_REPRESENTATION_SENSITIVITY": 5,
}


BATCHES = [
    1,
    64,
    256,
    1024,
    8192,
]


CPU_MODES = {
    "CPU_1_PHYSICAL_CORE",
    "CPU_2_PHYSICAL_CORE",
}


TARGETS = {
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
    "ENS_LGBM_XGB_EQUAL",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
}


PRIMARY = {
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
}


TPATH = {
    x: TDIR / f"{x}.csv"
    for x in TABLE_IDS
}


OUT = (
    ROOT
    / "stage26_8d5_cpu_publication_artifact_audit"
)

AUDIT = (
    OUT
    / "stage26_8d5_cpu_publication_artifact_audit.json"
)

RECEIPT = (
    OUT
    / "stage26_8d5_cpu_publication_artifact_audit_receipt.json"
)

MANIFEST = (
    OUT
    / "stage26_8d5_cpu_publication_artifact_audit_manifest.json"
)

COMMIT_MSG = (
    "stage26: audit CPU publication artifacts"
)


# =============================================================================
# HELPERS
# =============================================================================

def banner(text):
    print("\n" + "=" * 124)
    print(text)
    print("=" * 124)


def run(cmd, *, env=None, check=True):

    p = subprocess.run(
        cmd,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        check=False,
    )

    if check and p.returncode:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(cmd)}\n{p.stdout}"
        )

    return p.stdout.strip()


def git(*args, env=None):

    return run(
        [
            "git",
            *args,
        ],
        env=env,
    )


def sha(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                8 * 1024 * 1024
            ),
            b"",
        ):
            h.update(
                block
            )

    return h.hexdigest()


def jread(path):

    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as f:

        return json.load(
            f
        )


def cread(path):

    with Path(path).open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:

        reader = csv.DictReader(
            f
        )

        return (
            reader.fieldnames,
            list(
                reader
            ),
        )


def blank(value):

    return (
        str(
            value
        ).strip()
        ==
        ""
    )


def iv(value, label):

    try:
        return int(
            str(
                value
            ).strip()
        )

    except Exception as exc:
        raise RuntimeError(
            f"Invalid int {label}: {value!r}"
        ) from exc


def fv(value, label):

    try:
        return float(
            str(
                value
            ).strip()
        )

    except Exception as exc:
        raise RuntimeError(
            f"Invalid float {label}: {value!r}"
        ) from exc


def atomic_json(
    path,
    obj,
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = Path(
        str(
            path
        )
        +
        ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )

        f.write(
            "\n"
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp,
        path,
    )


def blob(
    ref,
    rel,
):

    p = subprocess.run(
        [
            "git",
            "show",
            f"{ref}:{rel}",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode:

        raise RuntimeError(
            p.stderr.decode(
                "utf-8",
                errors="replace",
            )
        )

    return p.stdout


def check_manifest(
    manifest,
    label,
):

    output = []
    seen = set()

    for item in manifest[
        "files"
    ]:

        rel = item[
            "path"
        ]

        if rel in seen:
            raise RuntimeError(
                f"Duplicate {label} manifest path: {rel}"
            )

        seen.add(
            rel
        )

        path = (
            REPO
            / rel
        )

        if not path.is_file():
            raise FileNotFoundError(
                path
            )

        actual_sha = sha(
            path
        )

        actual_size = (
            path
            .stat()
            .st_size
        )

        if (
            actual_sha
            !=
            item[
                "sha256"
            ]
            or
            actual_size
            !=
            int(
                item[
                    "size_bytes"
                ]
            )
        ):

            raise RuntimeError(
                f"{label} manifest mismatch: {rel}"
            )

        output.append(
            rel
        )

    return output


# =============================================================================
# CORRECTED TOKEN-AWARE CI DETECTOR
# =============================================================================

def is_ci_column(
    name: str,
) -> bool:

    n = (
        name
        .strip()
        .lower()
    )

    patterns = (
        "_ci_",
        "_ci95_",
        "_ci90_",
        "_ci99_",
        "ci_low",
        "ci_high",
        "ci95_low",
        "ci95_high",
        "ci90_low",
        "ci90_high",
        "ci99_low",
        "ci99_high",
        "confidence_interval",
        "confidence_low",
        "confidence_high",
    )

    if n == "ci":
        return True

    if n.startswith(
        "ci_"
    ):
        return True

    if n.endswith(
        "_ci"
    ):
        return True

    return any(
        pattern in n
        for pattern in patterns
    )


# =============================================================================
# DURABLE STATE GATE
# =============================================================================

banner(
    "STAGE26-8D5 :: DURABLE STATE GATE"
)

git(
    "fetch",
    "origin",
    "main",
)

head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)

print(
    "Expected parent:",
    EXPECTED_PARENT,
)

print(
    "Local HEAD     :",
    head,
)

print(
    "origin/main    :",
    remote,
)

print(
    "Repo clean     :",
    status == "",
)


if (
    head
    !=
    EXPECTED_PARENT
    or
    remote
    !=
    EXPECTED_PARENT
    or
    status
):

    raise RuntimeError(
        "Stage26-8D5 durable state gate failed."
    )


if OUT.exists():

    raise RuntimeError(
        f"8D5 output already exists: {OUT}"
    )


# =============================================================================
# SCHEMA / PROTOCOL GATE
# =============================================================================

banner(
    "STAGE26-8D5 :: SCHEMA / PROTOCOL GATE"
)


if sha(
    SCHEMA
) != SCHEMA_SHA:

    raise RuntimeError(
        "Effective schema SHA mismatch."
    )


schema = jread(
    SCHEMA
)


if (
    schema[
        "measurement_protocol_sha256"
    ]
    !=
    PROTOCOL_SHA
):

    raise RuntimeError(
        "Protocol SHA mismatch."
    )


if (
    schema[
        "publication_label"
    ]
    !=
    PUB_LABEL
):

    raise RuntimeError(
        "Publication label mismatch."
    )


if [
    item[
        "id"
    ]
    for item in schema[
        "tables"
    ]
] != TABLE_IDS:

    raise RuntimeError(
        "Table schema IDs changed."
    )


if [
    item[
        "id"
    ]
    for item in schema[
        "figures"
    ]
] != FIGURE_IDS:

    raise RuntimeError(
        "Figure schema IDs changed."
    )


scope = schema[
    "scope"
]


for key in [
    "gpu_allowed",
    "complete_e2e_measurement_available",
    "cross_group_pareto_allowed",
    "missing_cost_imputation_allowed",
]:

    if scope[
        key
    ] is not False:

        raise RuntimeError(
            f"Unsafe schema scope: {key}"
        )


print(
    "Protocol SHA :",
    PROTOCOL_SHA,
)

print(
    "Schema SHA   :",
    SCHEMA_SHA,
)

print(
    "Tables       : 8"
)

print(
    "Figures      : 8"
)

print(
    "GPU allowed  : False"
)


# =============================================================================
# CONTROL FILE IDENTITIES
# =============================================================================

banner(
    "STAGE26-8D5 :: CONTROL-FILE IDENTITY"
)


for path, expected in EXPECTED_CONTROL_SHA.items():

    actual = sha(
        path
    )

    print(
        f"{path.name}\n"
        f"  expected={expected}\n"
        f"  actual  ={actual}"
    )

    if actual != expected:

        raise RuntimeError(
            f"Control SHA mismatch: {path}"
        )


table_manifest = jread(
    TMANIFEST
)

figure_manifest = jread(
    FMANIFEST
)


table_manifest_paths = check_manifest(
    table_manifest,
    "8D3",
)

figure_manifest_paths = check_manifest(
    figure_manifest,
    "8D4",
)


if len(
    table_manifest_paths
) != 10:

    raise RuntimeError(
        "8D3 manifest expected 10 entries."
    )


if len(
    figure_manifest_paths
) != 18:

    raise RuntimeError(
        "8D4 manifest expected 18 entries."
    )


print(
    "8D3 manifest entries: 10 PASS"
)

print(
    "8D4 manifest entries: 18 PASS"
)


# =============================================================================
# HISTORICAL BYTE IDENTITY
# =============================================================================

banner(
    "STAGE26-8D5 :: HISTORICAL BYTE IDENTITY"
)


table_package = [
    *(
        TPATH[
            table_id
        ]
        for table_id in TABLE_IDS
    ),
    TINDEX,
    TRECEIPT,
    TMANIFEST,
]


figure_package = [
    *(
        FDIR
        /
        f"{figure_id}.{extension}"

        for figure_id in FIGURE_IDS

        for extension in (
            "png",
            "pdf",
        )
    ),
    FINDEX,
    FRECEIPT,
    FMANIFEST,
]


for (
    label,
    anchor,
    paths,
) in [

    (
        "8D3",
        TABLE_ANCHOR,
        table_package,
    ),

    (
        "8D4",
        FIGURE_ANCHOR,
        figure_package,
    ),
]:

    for path in paths:

        rel = str(
            path.relative_to(
                REPO
            )
        )

        ok = (
            path.read_bytes()
            ==
            blob(
                anchor,
                rel,
            )
        )

        print(
            f"{'PASS' if ok else 'FAIL'} "
            f"{label} {rel}"
        )

        if not ok:

            raise RuntimeError(
                f"{label} byte identity failed: {rel}"
            )


# =============================================================================
# LOAD TABLES
# =============================================================================

banner(
    "STAGE26-8D5 :: LOAD TABLES"
)


HEADERS = {}
TABLES = {}


for table_id in TABLE_IDS:

    (
        HEADERS[
            table_id
        ],
        TABLES[
            table_id
        ],
    ) = cread(
        TPATH[
            table_id
        ]
    )

    print(
        f"{table_id:52s} "
        f"rows={len(TABLES[table_id]):4d}"
    )

    if (
        len(
            TABLES[
                table_id
            ]
        )
        !=
        ROWS[
            table_id
        ]
    ):

        raise RuntimeError(
            f"{table_id} row count changed."
        )


# =============================================================================
# 8D3 RECEIPT
# =============================================================================

table_receipt = jread(
    TRECEIPT
)


if (
    table_receipt[
        "effective_schema_sha256"
    ]
    !=
    SCHEMA_SHA
):

    raise RuntimeError(
        "8D3 schema link mismatch."
    )


if (
    table_receipt[
        "table_row_counts"
    ]
    !=
    ROWS
):

    raise RuntimeError(
        "8D3 receipt row map mismatch."
    )


if (
    table_receipt[
        "complete_e2e"
    ]
    !=
    "UNAVAILABLE"
):

    raise RuntimeError(
        "8D3 E2E policy mismatch."
    )


if (
    table_receipt[
        "historical_stage26_4c3"
    ]
    !=
    "RETAINED_AS_ORIGINAL_MEASUREMENT"
):

    raise RuntimeError(
        "4C3 retention mismatch."
    )


if (
    table_receipt[
        "stage26_8_representation_sensitivity"
    ]
    !=
    "DESCRIPTIVE_SEPARATE_NOT_REPLACEMENT"
):

    raise RuntimeError(
        "Sensitivity policy mismatch."
    )


for (
    key,
    value,
) in table_receipt[
    "scientific_safety"
].items():

    if (
        isinstance(
            value,
            bool,
        )
        and
        value is not False
    ):

        raise RuntimeError(
            "8D3 scientific-safety flag "
            f"unexpectedly true: {key}"
        )


# =============================================================================
# WARM TABLE AUDIT
# =============================================================================

banner(
    "STAGE26-8D5 :: WARM TABLE AUDIT"
)


warm = TABLES[
    "T26_CPU_WARM_INFERENCE"
]


expected_warm_header = [
    "target_id",
    "group",
    "cpu_mode",
    "batch_size",
    "status",
    "n_timed_observations",
    "p50_latency_ms",
    "p50_ci95_low_ms",
    "p50_ci95_high_ms",
    "p95_latency_ms",
    "p95_ci95_low_ms",
    "p95_ci95_high_ms",
    "p99_latency_ms",
    "p99_ci95_low_ms",
    "p99_ci95_high_ms",
    "median_throughput_samples_per_s",
    "throughput_ci95_low_samples_per_s",
    "throughput_ci95_high_samples_per_s",
    "resource_limit_outcome",
]


if (
    HEADERS[
        "T26_CPU_WARM_INFERENCE"
    ]
    !=
    expected_warm_header
):

    raise RuntimeError(
        "Warm header changed."
    )


status_counts = Counter(
    row[
        "status"
    ]
    for row in warm
)


if status_counts != Counter(
    {
        "PASS": 73,
        "RESOURCE_LIMIT_OOM": 2,
        "TIMEOUT_RESOURCE_LIMIT": 5,
    }
):

    raise RuntimeError(
        f"Warm status counts changed: {status_counts}"
    )


warm_keys = set()


warm_metrics = [
    name

    for name in HEADERS[
        "T26_CPU_WARM_INFERENCE"
    ]

    if (
        name.endswith(
            "_ms"
        )
        or
        "throughput"
        in
        name
    )
]


for row in warm:

    batch = iv(
        row[
            "batch_size"
        ],
        "warm batch",
    )

    key = (
        row[
            "target_id"
        ],
        row[
            "cpu_mode"
        ],
        batch,
    )


    if key in warm_keys:

        raise RuntimeError(
            f"Duplicate warm key {key}"
        )


    warm_keys.add(
        key
    )


    if (
        row[
            "target_id"
        ]
        not in
        TARGETS
        or
        row[
            "cpu_mode"
        ]
        not in
        CPU_MODES
        or
        batch
        not in
        BATCHES
    ):

        raise RuntimeError(
            f"Unexpected warm geometry: {key}"
        )


    n = iv(
        row[
            "n_timed_observations"
        ],
        f"{key} n",
    )


    if row[
        "status"
    ] == "PASS":

        if (
            n <= 0
            or
            row[
                "resource_limit_outcome"
            ]
            !=
            "NONE"
        ):

            raise RuntimeError(
                f"Bad PASS row {key}"
            )


        for field in [
            "p50_latency_ms",
            "p50_ci95_low_ms",
            "p50_ci95_high_ms",
            "p95_latency_ms",
            "p95_ci95_low_ms",
            "p95_ci95_high_ms",
            "median_throughput_samples_per_s",
            "throughput_ci95_low_samples_per_s",
            "throughput_ci95_high_samples_per_s",
        ]:

            fv(
                row[
                    field
                ],
                f"{key} {field}",
            )


        p99_fields = [
            "p99_latency_ms",
            "p99_ci95_low_ms",
            "p99_ci95_high_ms",
        ]


        if n >= 100:

            for field in p99_fields:

                fv(
                    row[
                        field
                    ],
                    f"{key} {field}",
                )

        else:

            if any(
                not blank(
                    row[
                        field
                    ]
                )

                for field in p99_fields
            ):

                raise RuntimeError(
                    f"p99 leaked for n<100 {key}"
                )


    else:

        if (
            n != 0
            or
            row[
                "resource_limit_outcome"
            ]
            !=
            row[
                "status"
            ]
        ):

            raise RuntimeError(
                f"Bad non-PASS row {key}"
            )


        if any(
            not blank(
                row[
                    field
                ]
            )

            for field in warm_metrics
        ):

            raise RuntimeError(
                f"Non-PASS metric imputation detected {key}"
            )


expected_warm_keys = {
    (
        target,
        cpu_mode,
        batch,
    )

    for target in TARGETS

    for cpu_mode in CPU_MODES

    for batch in BATCHES
}


if (
    warm_keys
    !=
    expected_warm_keys
):

    raise RuntimeError(
        "Warm 8x2x5 geometry changed."
    )


print(
    "73 PASS / 2 OOM / 5 TIMEOUT: PASS"
)

print(
    "Corrected-CI shape             : PASS"
)

print(
    "Resource-limit imputation      : NONE"
)


# =============================================================================
# MEMORY TABLE AUDIT
# =============================================================================

banner(
    "STAGE26-8D5 :: MEMORY TABLE AUDIT"
)


memory = TABLES[
    "T26_CPU_MEMORY_PACKAGE"
]


memory_metric_set = {
    "median_baseline_rss_mib",
    "median_loaded_rss_mib",
    "median_peak_rss_mib",
    "median_delta_model_rss_mib",
    "median_delta_peak_rss_mib",
}


if {
    row[
        "memory_metric_name"
    ]
    for row in memory
} != memory_metric_set:

    raise RuntimeError(
        "Memory metric set changed."
    )


condition_metrics = defaultdict(
    set
)

package_values = defaultdict(
    set
)

memory_keys = set()


for row in memory:

    batch = iv(
        row[
            "batch_size"
        ],
        "memory batch",
    )


    key = (
        row[
            "target_id"
        ],
        row[
            "cpu_mode"
        ],
        batch,
        row[
            "memory_metric_name"
        ],
    )


    if key in memory_keys:

        raise RuntimeError(
            f"Duplicate memory key {key}"
        )


    memory_keys.add(
        key
    )


    condition = (
        key[
            0
        ],
        key[
            1
        ],
        key[
            2
        ],
    )


    condition_metrics[
        condition
    ].add(
        key[
            3
        ]
    )


    package_values[
        row[
            "target_id"
        ]
    ].add(
        row[
            "deployment_package_size_mib"
        ]
    )


    planned = iv(
        row[
            "planned_repetitions"
        ],
        f"{condition} planned",
    )

    passed = iv(
        row[
            "pass_repetitions"
        ],
        f"{condition} pass",
    )

    oom = iv(
        row[
            "resource_limit_oom_repetitions"
        ],
        f"{condition} oom",
    )

    timeout = iv(
        row[
            "timeout_resource_limit_repetitions"
        ],
        f"{condition} timeout",
    )


    if (
        planned != 5
        or
        passed
        +
        oom
        +
        timeout
        !=
        planned
    ):

        raise RuntimeError(
            f"Memory repetition accounting failed {condition}"
        )


    if passed > 0:

        fv(
            row[
                "memory_value_mib"
            ],
            f"{key} value",
        )

    elif not blank(
        row[
            "memory_value_mib"
        ]
    ):

        raise RuntimeError(
            f"Memory value imputed {key}"
        )


if (
    len(
        condition_metrics
    )
    !=
    80
    or
    any(
        metrics
        !=
        memory_metric_set

        for metrics in condition_metrics.values()
    )
):

    raise RuntimeError(
        "Memory condition geometry changed."
    )


if any(
    len(
        values
    )
    !=
    1

    for values in package_values.values()
):

    raise RuntimeError(
        "Package size varies by condition."
    )


print(
    "400 rows = 80 conditions x 5 metrics: PASS"
)

print(
    "CPU/batch aggregation                : NONE"
)

print(
    "Unavailable-memory imputation        : NONE"
)


# =============================================================================
# COLD / CAPACITY / COMPONENT / RATIO / PARETO / SENSITIVITY
# =============================================================================

banner(
    "STAGE26-8D5 :: "
    "COLD / CAPACITY / COMPONENT / RATIO / PARETO / SENSITIVITY"
)


# -----------------------------------------------------------------------------
# COLD START
# -----------------------------------------------------------------------------

cold = TABLES[
    "T26_CPU_COLD_START"
]


if any(
    is_ci_column(
        column
    )

    for column in HEADERS[
        "T26_CPU_COLD_START"
    ]
):

    raise RuntimeError(
        "Cold CI column leaked."
    )


if {
    row[
        "target_id"
    ]
    for row in cold
} != TARGETS:

    raise RuntimeError(
        "Cold target universe changed."
    )


cold_counts = Counter(
    row[
        "target_id"
    ]
    for row in cold
)


if set(
    cold_counts.values()
) != {
    7
}:

    raise RuntimeError(
        f"Cold component count changed: {cold_counts}"
    )


for row in cold:

    if (
        row[
            "cpu_mode"
        ]
        !=
        "CPU_1_PHYSICAL_CORE"
        or
        row[
            "status"
        ]
        !=
        "PASS"
        or
        iv(
            row[
                "n_observations"
            ],
            "cold n",
        )
        !=
        20
    ):

        raise RuntimeError(
            "Cold-start row invariant failed."
        )


    fv(
        row[
            "p50_ms"
        ],
        "cold p50",
    )

    fv(
        row[
            "p95_ms"
        ],
        "cold p95",
    )


print(
    "Cold-start: 56 point-estimate rows; CI ABSENT: PASS"
)


# -----------------------------------------------------------------------------
# CAPACITY
# -----------------------------------------------------------------------------

capacity = TABLES[
    "T26_CPU_CAPACITY_SCALING"
]


if any(
    is_ci_column(
        column
    )

    for column in HEADERS[
        "T26_CPU_CAPACITY_SCALING"
    ]
):

    raise RuntimeError(
        "Capacity CI leaked."
    )


capacity_keys = {
    (
        row[
            "target_id"
        ],
        iv(
            row[
                "batch_size"
            ],
            "capacity batch",
        ),
    )

    for row in capacity
}


if (
    capacity_keys
    !=
    {
        (
            target,
            batch,
        )

        for target in TARGETS

        for batch in BATCHES
    }
):

    raise RuntimeError(
        "Capacity geometry changed."
    )


print(
    "Capacity: 40 immutable target x batch rows; "
    "ratio CI ABSENT: PASS"
)


# -----------------------------------------------------------------------------
# COMPONENT BOUNDARIES
# -----------------------------------------------------------------------------

components = TABLES[
    "T26_COMPONENT_MEASUREMENTS"
]


if {
    row[
        "component_family"
    ]
    for row in components
} != {
    "RAW_EXTRACTION_COMPONENT",
    "PACKET_IMAGE_REPRESENTATION_COMPONENT",
    "ISOLATED_WARM_INFERENCE_COMPONENT",
    "COMPLETE_E2E",
}:

    raise RuntimeError(
        "Component family set changed."
    )


e2e_rows = [
    row

    for row in components

    if row[
        "component_family"
    ]
    ==
    "COMPLETE_E2E"
]


if (
    len(
        e2e_rows
    )
    !=
    1
    or
    e2e_rows[
        0
    ][
        "measurement_status"
    ]
    !=
    "CLOSED_NO_VALID_COMPLETE_E2E_MEASUREMENT"
    or
    e2e_rows[
        0
    ][
        "complete_e2e_available"
    ].lower()
    !=
    "false"
    or
    e2e_rows[
        0
    ][
        "quantitative_detail_reference"
    ]
    !=
    "NONE"
):

    raise RuntimeError(
        "Complete-E2E closure row changed."
    )


print(
    "Component boundaries: "
    "3 measured components + E2E unavailable: PASS"
)


# -----------------------------------------------------------------------------
# GROUP-B RATIO
# -----------------------------------------------------------------------------

ratio = TABLES[
    "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO"
]


expected_ratio_header = [
    "target_id",
    "batch_size",
    "inference_over_representation_capacity_ratio",
    "publication_label",
]


if (
    HEADERS[
        "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO"
    ]
    !=
    expected_ratio_header
):

    raise RuntimeError(
        "Group-B ratio direction/header changed."
    )


if any(
    is_ci_column(
        column
    )

    for column in HEADERS[
        "T26_GROUP_B_REPRESENTATION_INFERENCE_RATIO"
    ]
):

    raise RuntimeError(
        "Group-B ratio CI leaked."
    )


ratio_keys = {
    (
        row[
            "target_id"
        ],
        iv(
            row[
                "batch_size"
            ],
            "ratio batch",
        ),
    )

    for row in ratio
}


expected_ratio_keys = {
    (
        "STAGE20_MASKED_CNN_V1",
        1,
    ),
    (
        "STAGE20_MASKED_CNN_V1",
        64,
    ),
    (
        "STAGE20_MASKED_CNN_V1",
        256,
    ),
    (
        "STAGE21_MASKED_VIT_V1",
        1,
    ),
    (
        "STAGE21_MASKED_VIT_V1",
        64,
    ),
    (
        "STAGE21_MASKED_VIT_V1",
        256,
    ),
    (
        "STAGE21_MASKED_VIT_V1",
        1024,
    ),
}


if (
    ratio_keys
    !=
    expected_ratio_keys
):

    raise RuntimeError(
        "Group-B ratio matched-PASS geometry changed."
    )


for row in ratio:

    if (
        fv(
            row[
                "inference_over_representation_capacity_ratio"
            ],
            "ratio",
        )
        <=
        0
        or
        row[
            "publication_label"
        ]
        !=
        PUB_LABEL
    ):

        raise RuntimeError(
            "Group-B ratio row invariant failed."
        )


print(
    "Group-B inference/representation ratio: "
    "7 rows; CI ABSENT: PASS"
)


# -----------------------------------------------------------------------------
# PARETO
# -----------------------------------------------------------------------------

pareto = TABLES[
    "T26_PARETO"
]


if {
    row[
        "target_id"
    ]
    for row in pareto
} != PRIMARY:

    raise RuntimeError(
        "Pareto target universe changed."
    )


if Counter(
    row[
        "group"
    ]
    for row in pareto
) != Counter(
    {
        "GROUP_A_DUPSAFE70": 4,
        "GROUP_B_PACKET_IMAGE": 2,
    }
):

    raise RuntimeError(
        "Pareto group separation changed."
    )


pareto_membership = Counter(
    (
        row[
            "group"
        ],
        row[
            "pareto_status"
        ],
    )

    for row in pareto
)


if pareto_membership != Counter(
    {
        (
            "GROUP_A_DUPSAFE70",
            "FRONTIER_MEMBER",
        ): 3,

        (
            "GROUP_A_DUPSAFE70",
            "NOT_FRONTIER_MEMBER",
        ): 1,

        (
            "GROUP_B_PACKET_IMAGE",
            "FRONTIER_MEMBER",
        ): 1,

        (
            "GROUP_B_PACKET_IMAGE",
            "NOT_FRONTIER_MEMBER",
        ): 1,
    }
):

    raise RuntimeError(
        "Pareto membership changed."
    )


if any(
    is_ci_column(
        column
    )

    for column in HEADERS[
        "T26_PARETO"
    ]
):

    raise RuntimeError(
        "Pareto CI leaked."
    )


print(
    "Pareto: within-group immutable membership; "
    "PR-AUC CI ABSENT: PASS"
)


# -----------------------------------------------------------------------------
# REPRESENTATION SENSITIVITY
# -----------------------------------------------------------------------------

sensitivity = TABLES[
    "T26_REPRESENTATION_SENSITIVITY"
]


if [
    iv(
        row[
            "batch_size"
        ],
        "sensitivity batch",
    )

    for row in sensitivity
] != BATCHES:

    raise RuntimeError(
        "Sensitivity batches changed."
    )


for row in sensitivity:

    batch = iv(
        row[
            "batch_size"
        ],
        "sensitivity batch",
    )


    if (
        iv(
            row[
                "n_pairs"
            ],
            f"B{batch} pairs",
        )
        !=
        5
        or
        iv(
            row[
                "fingerprints_equal_count"
            ],
            f"B{batch} fingerprints",
        )
        !=
        5
    ):

        raise RuntimeError(
            f"Sensitivity geometry failed B{batch}"
        )


    if (
        fv(
            row[
                "median_throughput_ratio_v3_over_v2"
            ],
            f"B{batch} ratio",
        )
        <=
        0
    ):

        raise RuntimeError(
            f"Sensitivity ratio invalid B{batch}"
        )


print(
    "Sensitivity: "
    "5 batches / 25 pairs / 25 fingerprint matches: PASS"
)


# =============================================================================
# FIGURE PACKAGE AUDIT
# =============================================================================

banner(
    "STAGE26-8D5 :: "
    "FIGURE INDEX / RECEIPT / FILE STRUCTURE"
)


figure_index = jread(
    FINDEX
)

figure_receipt = jread(
    FRECEIPT
)


if (
    figure_index[
        "effective_schema_sha256"
    ]
    !=
    SCHEMA_SHA
    or
    figure_index[
        "publication_label"
    ]
    !=
    PUB_LABEL
):

    raise RuntimeError(
        "Figure index schema/publication link failed."
    )


if (
    figure_index[
        "quantitative_input_policy"
    ]
    !=
    "STAGE26_8D3_PUBLICATION_TABLES_ONLY"
):

    raise RuntimeError(
        "Figure quantitative input policy changed."
    )


if [
    figure[
        "figure_id"
    ]

    for figure in figure_index[
        "figures"
    ]
] != FIGURE_IDS:

    raise RuntimeError(
        "Figure index IDs changed."
    )


if (
    figure_index[
        "logical_figure_count"
    ]
    !=
    8
    or
    figure_index[
        "rendered_file_count"
    ]
    !=
    16
):

    raise RuntimeError(
        "Figure count changed."
    )


required_tokens = {

    "F26_WARM_LATENCY": {
        "CORRECTED_STAGE26_6F1_UNCERTAINTY_ONLY",
        "NO_STAGE26_2_HISTORICAL_CI",
        "RESOURCE_LIMITS_EXPLICIT",
        "NO_INTERPOLATION_THROUGH_RESOURCE_LIMITS",
    },

    "F26_WARM_THROUGHPUT": {
        "CORRECTED_STAGE26_6F1_UNCERTAINTY_ONLY",
        "RESOURCE_LIMITS_EXPLICIT",
        "NO_POST_HOC_BEST_BATCH",
    },

    "F26_MEMORY_PACKAGE": {
        "PACKAGE_SIZE_INDEPENDENT_PANEL",
        "ALL_FIVE_FROZEN_MEMORY_METRICS",
        "NO_CPU_OR_BATCH_AGGREGATION",
        "RAM_LIMITS_EXPLICIT",
        "NO_IMPUTATION",
    },

    "F26_COLD_START": {
        "POINT_ESTIMATES_ONLY",
        "NO_COLD_START_CI",
    },

    "F26_CAPACITY_SCALING": {
        "STAGE26_7C_COPIED_NOT_RECOMPUTED",
        "NO_RATIO_CI",
        "NO_POST_HOC_BEST_BATCH",
        "COMPONENT_LEVEL_ONLY",
    },

    "F26_PARETO": {
        "GROUPS_SEPARATE",
        "STAGE26_6D_MEMBERSHIP_COPIED",
        "NO_CROSS_GROUP_FRONTIER",
        "NO_PR_AUC_BOOTSTRAP",
    },

    "F26_REPRESENTATION_SENSITIVITY": {
        "DESCRIPTIVE_ONLY",
        "NEUTRAL_RATIO_ONE_REFERENCE",
        "NO_SIGNIFICANCE_TEST",
        "NO_FIXED_BIAS_CLAIM",
        "V3_NOT_REPLACEMENT",
    },

    "F26_COMPONENT_BOUNDARY": {
        "DISTINCT_COMPONENTS",
        "COMPLETE_E2E_UNAVAILABLE",
        "NO_ADDITIVE_LATENCY",
        "NO_SYNTHETIC_PIPELINE_THROUGHPUT",
        "NO_MISSING_COST_IMPUTATION",
    },
}


indexed_figure_files = {}


for figure in figure_index[
    "figures"
]:

    figure_id = figure[
        "figure_id"
    ]


    if not required_tokens[
        figure_id
    ].issubset(
        set(
            figure[
                "validations"
            ]
        )
    ):

        raise RuntimeError(
            "Figure validation tokens missing: "
            f"{figure_id}"
        )


    if (
        {
            file_info[
                "format"
            ]
            for file_info in figure[
                "files"
            ]
        }
        !=
        {
            "PNG",
            "PDF",
        }
        or
        len(
            figure[
                "files"
            ]
        )
        !=
        2
    ):

        raise RuntimeError(
            f"Figure formats changed: {figure_id}"
        )


    for file_info in figure[
        "files"
    ]:

        indexed_figure_files[
            file_info[
                "path"
            ]
        ] = file_info


unsafe_figure_receipt_flags = [
    "bootstrap_executed",
    "cold_start_ci_rendered",
    "complete_e2e_measurement_available",
    "gpu_used",
    "inference_executed",
    "labels_accessed",
    "memory_condition_aggregation",
    "model_loaded",
    "pareto_recomputed",
    "pcap_accessed",
    "random_sampling_executed",
    "ratio_inverted",
    "ratio_recomputed",
    "raw_measurement_files_read",
    "stage26_2_historical_ci_rendered",
    "stage26_7c_recomputed",
    "timing_executed",
]


for key in unsafe_figure_receipt_flags:

    if (
        figure_receipt[
            key
        ]
        is not False
    ):

        raise RuntimeError(
            f"8D4 unsafe receipt flag: {key}"
        )


if (
    figure_receipt[
        "effective_schema_sha256"
    ]
    !=
    SCHEMA_SHA
    or
    figure_receipt[
        "quantitative_inputs"
    ]
    !=
    "STAGE26_8D3_PUBLICATION_TABLES_ONLY"
):

    raise RuntimeError(
        "8D4 receipt linkage failed."
    )


for (
    table_id,
    expected,
) in figure_receipt[
    "table_sha256"
].items():

    if (
        sha(
            TPATH[
                table_id
            ]
        )
        !=
        expected
    ):

        raise RuntimeError(
            "8D4 input table hash drifted: "
            f"{table_id}"
        )


manifest_figure_files = {
    item[
        "path"
    ]:
        item

    for item in figure_manifest[
        "files"
    ]

    if item[
        "path"
    ].lower().endswith(
        (
            ".png",
            ".pdf",
        )
    )
}


if (
    set(
        manifest_figure_files
    )
    !=
    set(
        indexed_figure_files
    )
):

    raise RuntimeError(
        "Figure index/manifest file set mismatch."
    )


for (
    rel,
    indexed_info,
) in indexed_figure_files.items():

    path = (
        REPO
        / rel
    )

    manifest_info = manifest_figure_files[
        rel
    ]


    if (
        indexed_info[
            "sha256"
        ]
        !=
        manifest_info[
            "sha256"
        ]
        or
        int(
            indexed_info[
                "size_bytes"
            ]
        )
        !=
        int(
            manifest_info[
                "size_bytes"
            ]
        )
    ):

        raise RuntimeError(
            "Figure index/manifest metadata mismatch: "
            f"{rel}"
        )


    prefix = path.read_bytes()[
        :8
    ]


    if (
        rel.endswith(
            ".png"
        )
        and
        prefix
        !=
        b"\x89PNG\r\n\x1a\n"
    ):

        raise RuntimeError(
            f"Bad PNG signature: {rel}"
        )


    if (
        rel.endswith(
            ".pdf"
        )
        and
        not prefix.startswith(
            b"%PDF-"
        )
    ):

        raise RuntimeError(
            f"Bad PDF signature: {rel}"
        )


    if (
        path
        .stat()
        .st_size
        <
        1000
    ):

        raise RuntimeError(
            f"Rendered figure too small: {rel}"
        )


print(
    "8 figure families / 16 rendered files: PASS"
)

print(
    "Required validation tokens            : PASS"
)

print(
    "PNG/PDF signatures                    : PASS"
)

print(
    "8D3-table-only quantitative input     : PASS"
)


# =============================================================================
# CROSS-PACKAGE PROTOCOL VERDICT
# =============================================================================

banner(
    "STAGE26-8D5 :: CROSS-PACKAGE PROTOCOL VERDICT"
)


rules = schema[
    "global_rules"
]


if (
    "Point estimates only"
    not in
    rules[
        "cold_start_uncertainty"
    ]
):

    raise RuntimeError(
        "Cold rule missing."
    )


if (
    "Stage26-6F1"
    not in
    rules[
        "warm_cpu_uncertainty"
    ]
):

    raise RuntimeError(
        "Warm CI rule missing."
    )


if (
    "unavailable"
    not in
    rules[
        "complete_e2e"
    ].lower()
):

    raise RuntimeError(
        "E2E rule missing."
    )


if (
    "Never recompute"
    not in
    rules[
        "pareto"
    ]
):

    raise RuntimeError(
        "Pareto rule missing."
    )


if (
    "Retain historical Stage26-4C3"
    not in
    rules[
        "representation"
    ]
):

    raise RuntimeError(
        "4C3 rule missing."
    )


print(
    "Cold-start uncertainty : POINT ESTIMATES ONLY"
)

print(
    "Warm uncertainty       : STAGE26-6F1 CORRECTED ONLY"
)

print(
    "Memory aggregation     : NONE"
)

print(
    "Stage26-7C             : UNCHANGED"
)

print(
    "Ratio recomputation    : NONE"
)

print(
    "Ratio CI               : NONE"
)

print(
    "Pareto recomputation   : NONE"
)

print(
    "Cross-group Pareto     : NONE"
)

print(
    "PR-AUC bootstrap CI    : NONE"
)

print(
    "Historical Stage26-4C3 : RETAINED"
)

print(
    "Stage26-8 sensitivity  : DESCRIPTIVE / SEPARATE"
)

print(
    "Complete E2E           : UNAVAILABLE"
)

print(
    "GPU                    : OFF"
)


# =============================================================================
# WRITE DURABLE AUDIT
# =============================================================================

banner(
    "STAGE26-8D5 :: WRITE DURABLE AUDIT"
)


OUT.mkdir(
    parents=True,
    exist_ok=False,
)


now = datetime.now(
    timezone.utc
).isoformat()


audit = {

    "schema":
        "stage26_8d5_cpu_publication_artifact_audit_v1",

    "created_at_utc":
        now,

    "scientific_parent":
        EXPECTED_PARENT,

    "measurement_protocol_sha256":
        PROTOCOL_SHA,

    "effective_schema_sha256":
        SCHEMA_SHA,

    "stage26_8d3_anchor":
        TABLE_ANCHOR,

    "stage26_8d4_anchor":
        FIGURE_ANCHOR,

    "table_count":
        8,

    "logical_figure_count":
        8,

    "rendered_figure_file_count":
        16,

    "table_row_counts":
        ROWS,

    "warm_status_counts": {
        "PASS":
            73,

        "RESOURCE_LIMIT_OOM":
            2,

        "TIMEOUT_RESOURCE_LIMIT":
            5,
    },

    "scientific_state": {

        "cold_start_publication":
            "POINT_ESTIMATES_ONLY",

        "warm_uncertainty":
            "STAGE26_6F1_CORRECTED_ONLY",

        "memory_aggregation":
            False,

        "stage26_7c_recomputed":
            False,

        "ratio_recomputed":
            False,

        "ratio_inverted":
            False,

        "ratio_ci":
            False,

        "pareto_recomputed":
            False,

        "cross_group_pareto":
            False,

        "pr_auc_bootstrap_ci":
            False,

        "historical_stage26_4c3":
            "RETAINED",

        "stage26_8_sensitivity":
            "DESCRIPTIVE_SEPARATE_NOT_REPLACEMENT",

        "complete_e2e":
            "UNAVAILABLE",
    },

    "audit_implementation": {

        "ci_column_detection":
            "TOKEN_AWARE",

        "naive_ci_substring_detection":
            False,

        "stage26_8d5r1_false_positive_resolved":
            True,
    },

    "non_computation": {

        "timing":
            False,

        "inference":
            False,

        "model_loading":
            False,

        "bootstrap":
            False,

        "random_sampling":
            False,

        "pcap":
            False,

        "labels":
            False,

        "gpu":
            False,
    },

    "verdict":
        "CPU_PUBLICATION_ARTIFACT_PACKAGE_PROTOCOL_CONFORMANT",

    "gpu_allowed_after_this_stage":
        False,

    "next_stage":
        (
            "STAGE26_8D6_FINAL_CPU_CLOSURE_AND_"
            "FRESH_SESSION_GPU_BOOTSTRAP_MANIFEST"
        ),
}


atomic_json(
    AUDIT,
    audit,
)


audit_sha = sha(
    AUDIT
)


receipt = {

    "schema":
        "stage26_8d5_cpu_publication_artifact_audit_receipt_v1",

    "created_at_utc":
        now,

    "scientific_parent":
        EXPECTED_PARENT,

    "measurement_protocol_sha256":
        PROTOCOL_SHA,

    "effective_schema_sha256":
        SCHEMA_SHA,

    "stage26_8d3_manifest_sha256":
        EXPECTED_CONTROL_SHA[
            TMANIFEST
        ],

    "stage26_8d4_manifest_sha256":
        EXPECTED_CONTROL_SHA[
            FMANIFEST
        ],

    "audit_sha256":
        audit_sha,

    "ci_column_detection":
        "TOKEN_AWARE",

    "verdict":
        "CPU_PUBLICATION_ARTIFACT_PACKAGE_PROTOCOL_CONFORMANT",

    "gpu_allowed":
        False,
}


atomic_json(
    RECEIPT,
    receipt,
)


receipt_sha = sha(
    RECEIPT
)


manifest = {

    "schema":
        "stage26_8d5_cpu_publication_artifact_audit_manifest_v1",

    "created_at_utc":
        now,

    "scientific_parent":
        EXPECTED_PARENT,

    "files": [

        {
            "path":
                str(
                    AUDIT.relative_to(
                        REPO
                    )
                ),

            "sha256":
                audit_sha,

            "size_bytes":
                AUDIT.stat().st_size,
        },

        {
            "path":
                str(
                    RECEIPT.relative_to(
                        REPO
                    )
                ),

            "sha256":
                receipt_sha,

            "size_bytes":
                RECEIPT.stat().st_size,
        },
    ],
}


atomic_json(
    MANIFEST,
    manifest,
)


manifest_sha = sha(
    MANIFEST
)


print(
    "Audit SHA256   :",
    audit_sha,
)

print(
    "Receipt SHA256 :",
    receipt_sha,
)

print(
    "Manifest SHA256:",
    manifest_sha,
)


# =============================================================================
# PRE-COMMIT GIT AUDIT
# =============================================================================

banner(
    "STAGE26-8D5 :: PRE-COMMIT GIT AUDIT"
)


lines = [
    line

    for line in git(
        "status",
        "--porcelain",
    ).splitlines()

    if line.strip()
]


for line in lines:
    print(
        line
    )


prefix = (
    "?? "
    +
    str(
        OUT.relative_to(
            REPO
        )
    )
)


if (
    not lines
    or
    any(
        not line.startswith(
            prefix
        )

        for line in lines
    )
):

    raise RuntimeError(
        "Unexpected pre-commit repository state."
    )


# =============================================================================
# COMMIT
# =============================================================================

banner(
    "STAGE26-8D5 :: COMMIT"
)


git(
    "add",
    str(
        OUT.relative_to(
            REPO
        )
    ),
)


staged = git(
    "diff",
    "--cached",
    "--name-only",
).splitlines()


expected_staged = sorted(
    str(
        path.relative_to(
            REPO
        )
    )

    for path in [
        AUDIT,
        RECEIPT,
        MANIFEST,
    ]
)


print(
    "Staged files:"
)


for path in staged:
    print(
        " ",
        path
    )


if sorted(
    staged
) != expected_staged:

    raise RuntimeError(
        "Unexpected staged file set."
    )


git(
    "commit",
    "-m",
    COMMIT_MSG,
)


new_head = git(
    "rev-parse",
    "HEAD",
)


parent = git(
    "rev-parse",
    "HEAD^",
)


subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print(
    "Parent :",
    parent,
)

print(
    "HEAD   :",
    new_head,
)

print(
    "Subject:",
    subject,
)


if (
    parent
    !=
    EXPECTED_PARENT
    or
    subject
    !=
    COMMIT_MSG
):

    raise RuntimeError(
        "8D5 commit identity failed."
    )


# =============================================================================
# PUSH
# =============================================================================

banner(
    "STAGE26-8D5 :: PUSH"
)


from kaggle_secrets import UserSecretsClient


token = (
    UserSecretsClient()
    .get_secret(
        "GITHUB_TOKEN"
    )
)


if (
    not token
    or
    len(
        token.strip()
    )
    <
    20
):

    raise RuntimeError(
        "GITHUB_TOKEN unavailable."
    )


fd, askpass_name = tempfile.mkstemp(
    prefix="stage26_8d5_askpass_",
    suffix=".sh",
)

os.close(
    fd
)


askpass = Path(
    askpass_name
)


try:

    askpass.write_text(
        "#!/bin/sh\n"
        "case \"$1\" in\n"
        "  *Username*) printf \"%s\\n\" \"x-access-token\" ;;\n"
        "  *Password*) printf \"%s\\n\" \"$STAGE26_GITHUB_TOKEN\" ;;\n"
        "  *) printf \"%s\\n\" \"\" ;;\n"
        "esac\n",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        |
        stat.S_IXUSR
        |
        stat.S_IXGRP
        |
        stat.S_IXOTH
    )


    env = os.environ.copy()


    env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )


    env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"


    env[
        "STAGE26_GITHUB_TOKEN"
    ] = token.strip()


    print(
        git(
            "push",
            "origin",
            "main",
            env=env,
        )
    )


finally:

    askpass.unlink(
        missing_ok=True
    )

    token = None

    if "env" in locals():

        env.pop(
            "STAGE26_GITHUB_TOKEN",
            None,
        )


# =============================================================================
# REMOTE BYTE VERIFICATION
# =============================================================================

banner(
    "STAGE26-8D5 :: REMOTE BYTE VERIFICATION"
)


git(
    "fetch",
    "origin",
    "main",
)


local_head = git(
    "rev-parse",
    "HEAD",
)


remote_head = git(
    "rev-parse",
    "origin/main",
)


print(
    "Local HEAD :",
    local_head,
)

print(
    "origin/main:",
    remote_head,
)


if (
    local_head
    !=
    new_head
    or
    remote_head
    !=
    new_head
):

    raise RuntimeError(
        "Local/remote commit mismatch."
    )


for path in [
    AUDIT,
    RECEIPT,
    MANIFEST,
]:

    rel = str(
        path.relative_to(
            REPO
        )
    )


    local_bytes = path.read_bytes()


    remote_bytes = blob(
        "origin/main",
        rel,
    )


    ok = (
        local_bytes
        ==
        remote_bytes
    )


    print(
        f"{'PASS' if ok else 'FAIL'} "
        f"{rel}"
    )


    print(
        "  local :",
        hashlib.sha256(
            local_bytes
        ).hexdigest(),
    )


    print(
        "  remote:",
        hashlib.sha256(
            remote_bytes
        ).hexdigest(),
    )


    if not ok:

        raise RuntimeError(
            f"Remote byte mismatch: {rel}"
        )


# =============================================================================
# COMPLETE
# =============================================================================

final_status = git(
    "status",
    "--porcelain",
)


banner(
    "STAGE26-8D5 COMPLETE"
)


print(
    "Durable audit anchor        :",
    new_head,
)

print(
    "HEAD == origin/main         :",
    (
        local_head
        ==
        remote_head
        ==
        new_head
    ),
)

print(
    "Repo clean                  :",
    final_status == "",
)

print(
    "Publication tables          : 8 / PASS"
)

print(
    "Publication figure families : 8 / PASS"
)

print(
    "Rendered figure files       : 16 / PASS"
)

print(
    "CI-column detector          : TOKEN-AWARE"
)

print(
    "Protocol verdict            : "
    "CPU_PUBLICATION_ARTIFACT_PACKAGE_PROTOCOL_CONFORMANT"
)

print(
    "GPU                          : OFF"
)

print(
    "GPU allowed yet              : NO"
)


print(
    "\nNEXT:"
)

print(
    "  Stage26-8D6 final CPU closure + "
    "fresh-session GPU bootstrap manifest."
)

print(
    "  Freeze exact repo/corpus/model/artifact recovery identities "
    "for the expected Kaggle GPU reset."
)


if final_status:

    raise RuntimeError(
        "Repository not clean after Stage26-8D5."
    )


STAGE26-8D5 :: DURABLE STATE GATE
Expected parent: aefaa993e86dbc1d7fbd7e9c301e629883b0c51d
Local HEAD     : aefaa993e86dbc1d7fbd7e9c301e629883b0c51d
origin/main    : aefaa993e86dbc1d7fbd7e9c301e629883b0c51d
Repo clean     : True

STAGE26-8D5 :: SCHEMA / PROTOCOL GATE
Protocol SHA : d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625
Schema SHA   : 973cc58ed6572d30b1cce94f3659226d99b185390c2aca92d9ed030e50358ad0
Tables       : 8
Figures      : 8
GPU allowed  : False

STAGE26-8D5 :: CONTROL-FILE IDENTITY
stage26_8d3_cpu_publication_tables_index.json
  expected=ae869df663d349a448f797789f285cdd6bfa6f7c98183ff30f4102a14ee374ea
  actual  =ae869df663d349a448f797789f285cdd6bfa6f7c98183ff30f4102a14ee374ea
stage26_8d3_cpu_publication_tables_receipt.json
  expected=941eca833d7adaa48227097be858ca82e74364ba84ce3c42162a4568dd0e3a04
  actual  =941eca833d7adaa48227097be858ca82e74364ba84ce3c42162a4568dd0e3a04
stage26_8d3_cpu_publication_tables_manifest.json
  expected=656b0e936a18405e05d

In [24]:
# =============================================================================
# STAGE26-8D6 :: FINAL CPU CLOSURE + FRESH-SESSION GPU BOOTSTRAP FREEZE
#
# FINAL CPU CELL.
#
# PURPOSE
# -------
# 1. Re-verify the durable Stage26-8D5 publication-artifact audit.
# 2. Confirm this is still the CPU-only session.
# 3. Freeze the complete exact parent-repository recovery inventory.
# 4. Freeze reset-sensitive runtime asset candidates without reading raw PCAP.
# 5. Freeze the GPU profiling/bootstrap contract for a fresh Kaggle session.
# 6. Commit, push, remotely byte-verify, and close the CPU phase.
#
# AFTER THIS CELL PASSES:
#   CPU_PHASE  = COMPLETE
#   GPU_ALLOWED = TRUE
#
# The next GPU session MUST clone the exact commit SHA emitted by this stage.
#
# THIS CELL DOES NOT:
#   - run timing
#   - run inference
#   - load models for inference
#   - bootstrap statistics
#   - retrain/convert models
#   - read/hash raw PCAP
#   - enable/use GPU
# =============================================================================

from __future__ import annotations

import hashlib
import json
import os
import re
import stat
import subprocess
import tempfile

from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit


# =============================================================================
# FROZEN CONSTANTS
# =============================================================================

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")
ROOT = REPO / "results" / "stage26_deployment_profiling"
RUNTIME_ROOT = Path("/kaggle/working/stage26_deployment_profiling")

EXPECTED_PARENT = "9c560a37d7a5d3bd79ad1a72a103e64c57548f26"

REPO_URL = "https://github.com/themubasshir/ids2018-validation-safe-ablation.git"
BRANCH = "main"

PROTOCOL_SHA = "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
CPU_PLAN_SHA = "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363"
PREFLIGHT_RECEIPT_SHA = "4c7fc55a0e44e53587d385989dfec4f3f57f20768c7dd4a2d08e2936d2ba21e5"
SCHEMA_SHA = "973cc58ed6572d30b1cce94f3659226d99b185390c2aca92d9ed030e50358ad0"

STAGE26_8D5_AUDIT_SHA = "e4375ab82ce31a9120038950c268397570cf7fb741339a843433417bfa279225"
STAGE26_8D5_RECEIPT_SHA = "f97f8396b1fa7eaa3315060007a3def4b4266ac3e1ca5dbf5d52bef74394fb42"
STAGE26_8D5_MANIFEST_SHA = "5def9c65b11adddb2243f83455f3531642a278f7ccd44ad285fd41879d03b03b"

SCHEMA_PATH = (
    ROOT
    / "stage26_8d2a_cpu_publication_schema_erratum"
    / "stage26_8d2a_effective_cpu_publication_schema.json"
)

AUDIT5_DIR = ROOT / "stage26_8d5_cpu_publication_artifact_audit"
AUDIT5 = AUDIT5_DIR / "stage26_8d5_cpu_publication_artifact_audit.json"
RECEIPT5 = AUDIT5_DIR / "stage26_8d5_cpu_publication_artifact_audit_receipt.json"
MANIFEST5 = AUDIT5_DIR / "stage26_8d5_cpu_publication_artifact_audit_manifest.json"

OUT = ROOT / "stage26_8d6_final_cpu_closure"

PARENT_INVENTORY = OUT / "stage26_8d6_parent_repository_inventory.json"
RUNTIME_INVENTORY = OUT / "stage26_8d6_runtime_recovery_inventory.json"
GPU_CONTRACT = OUT / "stage26_8d6_gpu_bootstrap_contract.json"
CLOSURE_RECEIPT = OUT / "stage26_8d6_final_cpu_closure_receipt.json"
CLOSURE_MANIFEST = OUT / "stage26_8d6_final_cpu_closure_manifest.json"

COMMIT_MSG = "stage26: close CPU profiling and freeze GPU bootstrap"

TARGETS = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
    "ENS_LGBM_XGB_EQUAL",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
]

BATCHES = [1, 64, 256, 1024, 8192]

WARMUP_TIMED = {
    "1": {"warmup": 50, "timed": 200},
    "64": {"warmup": 30, "timed": 150},
    "256": {"warmup": 20, "timed": 100},
    "1024": {"warmup": 10, "timed": 50},
    "8192": {"warmup": 5, "timed": 20},
}

SEED = 26042

RAW_PACKET_SUFFIXES = {
    ".pcap",
    ".pcapng",
    ".cap",
}

DATA_MODEL_SUFFIXES = {
    ".parquet",
    ".npz",
    ".npy",
    ".pt",
    ".pth",
    ".ckpt",
    ".cbm",
    ".ubj",
    ".joblib",
    ".pkl",
    ".pickle",
    ".onnx",
    ".safetensors",
    ".model",
    ".bin",
    ".csv",
}

ASSET_NAME_TOKENS = (
    "corpus",
    "compact",
    "dataset",
    "model",
    "checkpoint",
    "artifact",
    "stage16",
    "stage20",
    "stage21",
    "feature",
    "packet_image",
    "packet-image",
)

RUNTIME_HASH_MIN_BYTES = 1 * 1024 * 1024


# =============================================================================
# HELPERS
# =============================================================================

def banner(text: str) -> None:
    print("\n" + "=" * 124)
    print(text)
    print("=" * 124)


def run(cmd, *, cwd=REPO, env=None, check=True, text=True):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=text,
        env=env,
        check=False,
    )
    if check and p.returncode != 0:
        output = p.stdout if text else p.stdout.decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}\n{output}"
        )
    return p


def git(*args, env=None, check=True) -> str:
    return run(["git", *args], env=env, check=check).stdout.strip()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def read_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def atomic_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True, allow_nan=False)
        f.write("\n")
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def git_blob(ref: str, rel: str) -> bytes:
    p = subprocess.run(
        ["git", "show", f"{ref}:{rel}"],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )
    if p.returncode != 0:
        raise RuntimeError(
            p.stderr.decode("utf-8", errors="replace")
        )
    return p.stdout


def sanitize_git_url(url: str) -> str:
    url = url.strip()
    if "://" not in url:
        return url
    parts = urlsplit(url)
    host = parts.hostname or ""
    if parts.port:
        host = f"{host}:{parts.port}"
    return urlunsplit((parts.scheme, host, parts.path, parts.query, parts.fragment))


def git_ls_files_stage() -> list[dict]:
    p = subprocess.run(
        ["git", "ls-files", "-s", "-z"],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )
    if p.returncode != 0:
        raise RuntimeError(
            p.stderr.decode("utf-8", errors="replace")
        )

    out = []
    for raw in p.stdout.split(b"\0"):
        if not raw:
            continue
        meta, path_b = raw.split(b"\t", 1)
        mode_b, oid_b, stage_b = meta.split(b" ", 2)
        out.append(
            {
                "mode": mode_b.decode("ascii"),
                "git_object_id": oid_b.decode("ascii"),
                "stage": int(stage_b.decode("ascii")),
                "path": path_b.decode("utf-8", errors="surrogateescape"),
            }
        )
    return out


def materialized_bytes(path: Path, mode: str) -> bytes:
    if mode == "120000":
        return os.readlink(path).encode("utf-8", errors="surrogateescape")
    return path.read_bytes()


def parse_lfs_pointer(data: bytes):
    if not data.startswith(b"version https://git-lfs.github.com/spec/v1\n"):
        return None
    text = data.decode("utf-8", errors="replace")
    oid_m = re.search(r"^oid sha256:([0-9a-f]{64})$", text, flags=re.M)
    size_m = re.search(r"^size ([0-9]+)$", text, flags=re.M)
    return {
        "oid_sha256": oid_m.group(1) if oid_m else None,
        "object_size_bytes": int(size_m.group(1)) if size_m else None,
    }


def repo_asset_candidate(rel: str) -> bool:
    p = Path(rel)
    lower = rel.lower()
    return (
        p.suffix.lower() in DATA_MODEL_SUFFIXES
        or any(token in lower for token in ASSET_NAME_TOKENS)
    )


def runtime_asset_candidate(path: Path, size: int) -> bool:
    lower = path.name.lower()
    suffix = path.suffix.lower()
    return (
        suffix in DATA_MODEL_SUFFIXES
        or any(token in str(path).lower() for token in ASSET_NAME_TOKENS)
        or size >= RUNTIME_HASH_MIN_BYTES
    )


def optional_structure(path: Path):
    suffix = path.suffix.lower()

    if suffix == ".parquet":
        try:
            import pyarrow.parquet as pq
            meta = pq.ParquetFile(path).metadata
            return {
                "kind": "PARQUET",
                "row_count": int(meta.num_rows),
                "column_count": int(meta.num_columns),
                "row_group_count": int(meta.num_row_groups),
            }
        except Exception as exc:
            return {
                "kind": "PARQUET",
                "metadata_status": "UNAVAILABLE",
                "reason": type(exc).__name__,
            }

    if suffix == ".npy":
        try:
            import numpy as np
            arr = np.load(path, mmap_mode="r", allow_pickle=False)
            return {
                "kind": "NPY",
                "shape": [int(x) for x in arr.shape],
                "dtype": str(arr.dtype),
            }
        except Exception as exc:
            return {
                "kind": "NPY",
                "metadata_status": "UNAVAILABLE",
                "reason": type(exc).__name__,
            }

    if suffix == ".npz":
        try:
            import zipfile
            with zipfile.ZipFile(path, "r") as z:
                members = [
                    {
                        "name": info.filename,
                        "compressed_size_bytes": int(info.compress_size),
                        "uncompressed_size_bytes": int(info.file_size),
                    }
                    for info in z.infolist()
                ]
            return {
                "kind": "NPZ",
                "member_count": len(members),
                "members": members,
                "note": "ZIP_DIRECTORY_METADATA_ONLY_NO_ARRAY_DECOMPRESSION",
            }
        except Exception as exc:
            return {
                "kind": "NPZ",
                "metadata_status": "UNAVAILABLE",
                "reason": type(exc).__name__,
            }

    return None


# =============================================================================
# 1. DURABLE STATE + CPU-ONLY GATE
# =============================================================================

banner("STAGE26-8D6 :: DURABLE STATE + CPU-ONLY GATE")

git("fetch", "origin", "main")

head = git("rev-parse", "HEAD")
origin = git("rev-parse", "origin/main")
status = git("status", "--porcelain")

origin_url = sanitize_git_url(git("remote", "get-url", "origin"))

print("Expected parent :", EXPECTED_PARENT)
print("Local HEAD      :", head)
print("origin/main     :", origin)
print("Repo clean      :", status == "")
print("Origin URL      :", origin_url)

if head != EXPECTED_PARENT:
    raise RuntimeError("Unexpected local HEAD.")
if origin != EXPECTED_PARENT:
    raise RuntimeError("Unexpected origin/main.")
if status:
    raise RuntimeError("Repository is not clean.")
if origin_url.rstrip("/") != REPO_URL.rstrip("/"):
    raise RuntimeError(
        f"Unexpected origin URL: {origin_url!r}"
    )
if OUT.exists():
    raise RuntimeError(f"8D6 output already exists: {OUT}")

try:
    nvidia = run(
        ["nvidia-smi", "-L"],
        cwd=REPO,
        check=False,
    ).stdout.strip()
except FileNotFoundError:
    nvidia = ""

gpu_visible_nvidia = bool(nvidia) and "no devices were found" not in nvidia.lower()

try:
    import torch
    torch_cuda_available = bool(torch.cuda.is_available())
    torch_version = str(torch.__version__)
except Exception as exc:
    torch_cuda_available = False
    torch_version = f"IMPORT_UNAVAILABLE:{type(exc).__name__}"

print("nvidia-smi GPU visible :", gpu_visible_nvidia)
print("torch.cuda available  :", torch_cuda_available)
print("torch version          :", torch_version)

if gpu_visible_nvidia or torch_cuda_available:
    raise RuntimeError(
        "GPU is visible before final CPU closure. Keep accelerator OFF for Stage26-8D6."
    )


# =============================================================================
# 2. 8D5 / SCHEMA / PROTOCOL IDENTITY
# =============================================================================

banner("STAGE26-8D6 :: 8D5 / SCHEMA / PROTOCOL IDENTITY")

expected_controls = {
    AUDIT5: STAGE26_8D5_AUDIT_SHA,
    RECEIPT5: STAGE26_8D5_RECEIPT_SHA,
    MANIFEST5: STAGE26_8D5_MANIFEST_SHA,
    SCHEMA_PATH: SCHEMA_SHA,
}

for path, expected in expected_controls.items():
    actual = sha256_file(path)
    print(f"{path.name}")
    print(f"  expected={expected}")
    print(f"  actual  ={actual}")
    if actual != expected:
        raise RuntimeError(f"Control SHA mismatch: {path}")

audit5 = read_json(AUDIT5)
receipt5 = read_json(RECEIPT5)

if audit5["verdict"] != "CPU_PUBLICATION_ARTIFACT_PACKAGE_PROTOCOL_CONFORMANT":
    raise RuntimeError("8D5 audit verdict changed.")
if audit5["measurement_protocol_sha256"] != PROTOCOL_SHA:
    raise RuntimeError("8D5 protocol link changed.")
if audit5["effective_schema_sha256"] != SCHEMA_SHA:
    raise RuntimeError("8D5 schema link changed.")
if audit5["gpu_allowed_after_this_stage"] is not False:
    raise RuntimeError("8D5 GPU gate changed.")
if audit5["non_computation"]["gpu"] is not False:
    raise RuntimeError("8D5 unexpectedly records GPU use.")
if audit5["scientific_state"]["complete_e2e"] != "UNAVAILABLE":
    raise RuntimeError("8D5 E2E state changed.")

if receipt5["verdict"] != "CPU_PUBLICATION_ARTIFACT_PACKAGE_PROTOCOL_CONFORMANT":
    raise RuntimeError("8D5 receipt verdict changed.")
if receipt5["gpu_allowed"] is not False:
    raise RuntimeError("8D5 receipt GPU gate changed.")

schema = read_json(SCHEMA_PATH)

if schema["measurement_protocol_sha256"] != PROTOCOL_SHA:
    raise RuntimeError("Effective schema protocol SHA changed.")
if schema["scope"]["gpu_allowed"] is not False:
    raise RuntimeError("Frozen CPU schema must still say GPU not allowed inside CPU phase.")

print("8D5 verdict   : CPU_PUBLICATION_ARTIFACT_PACKAGE_PROTOCOL_CONFORMANT")
print("Protocol SHA  :", PROTOCOL_SHA)
print("CPU plan SHA  :", CPU_PLAN_SHA)
print("Schema SHA    :", SCHEMA_SHA)
print("Complete E2E  : UNAVAILABLE")
print("GPU used      : NO")


# =============================================================================
# 3. GIT LFS DISCOVERY
# =============================================================================

banner("STAGE26-8D6 :: GIT LFS DISCOVERY")

tracked_stage = git_ls_files_stage()

if any(x["stage"] != 0 for x in tracked_stage):
    raise RuntimeError("Non-stage-0 tracked index entries found.")

attribute_paths = [
    x["path"]
    for x in tracked_stage
    if Path(x["path"]).name == ".gitattributes"
]

lfs_patterns_declared = False
for rel in attribute_paths:
    text = (REPO / rel).read_text(encoding="utf-8", errors="replace")
    if "filter=lfs" in text:
        lfs_patterns_declared = True

lfs_version_proc = run(["git", "lfs", "version"], check=False)
git_lfs_available = lfs_version_proc.returncode == 0

lfs_entries = {}
if git_lfs_available:
    p = run(["git", "lfs", "ls-files", "-l"], check=False)
    if p.returncode == 0:
        for line in p.stdout.splitlines():
            line = line.strip()
            if not line:
                continue
            # Typical forms:
            # <64hex> * path
            # <64hex> - path
            m = re.match(r"^([0-9a-f]{64})\s+[\*\-]\s+(.*)$", line)
            if m:
                lfs_entries[m.group(2)] = m.group(1)

print("Tracked .gitattributes files :", len(attribute_paths))
print("LFS patterns declared        :", lfs_patterns_declared)
print("git-lfs available            :", git_lfs_available)
print("git-lfs tracked paths        :", len(lfs_entries))

if lfs_patterns_declared and not git_lfs_available:
    raise RuntimeError(
        "Repository declares Git LFS patterns but git-lfs is unavailable; "
        "fresh-session recovery cannot be frozen safely."
    )


# =============================================================================
# 4. EXACT PARENT REPOSITORY INVENTORY
# =============================================================================

banner("STAGE26-8D6 :: EXACT PARENT REPOSITORY INVENTORY")

repo_entries = []
repo_total_bytes = 0
repo_asset_candidates = []
lfs_pointer_count = 0

for idx, item in enumerate(tracked_stage, start=1):
    rel = item["path"]
    mode = item["mode"]
    path = REPO / rel

    # Raw packet captures are never read/hashed by this CPU-closure inventory.
    # Their Git object/LFS identity is sufficient for repository provenance,
    # and they are explicitly not part of compact-corpus reconstruction.
    if Path(rel).suffix.lower() in RAW_PACKET_SUFFIXES and mode != "160000":
        if not path.exists():
            raise RuntimeError(f"Tracked raw packet file not materialized: {rel}")
        size = path.stat().st_size
        lfs_oid = lfs_entries.get(rel)
        repo_entries.append(
            {
                **item,
                "object_type": "RAW_PACKET_FILE",
                "materialized": True,
                "size_bytes": size,
                "sha256_materialized": None,
                "content_hashing_skipped": True,
                "content_hashing_skip_reason": (
                    "RAW_PACKET_FILE_EXCLUDED_BY_FINAL_CPU_CLOSURE_POLICY"
                ),
                "lfs_tracked": rel in lfs_entries,
                "lfs_oid_sha256": lfs_oid,
                "lfs_pointer_materialized": None,
                "lfs_object_size_bytes_from_pointer": None,
                "recovery_candidate": False,
            }
        )
        repo_total_bytes += size
        continue

    if mode == "160000":
        # Submodule gitlink. Do not silently assume it is materialized.
        entry = {
            **item,
            "object_type": "GITLINK",
            "materialized": path.exists(),
            "size_bytes": None,
            "sha256_materialized": None,
            "lfs_tracked": False,
            "lfs_oid_sha256": None,
            "lfs_pointer_materialized": False,
        }
        repo_entries.append(entry)
        continue

    if mode == "120000":
        if not path.is_symlink():
            raise RuntimeError(f"Tracked symlink not materialized: {rel}")
        data = materialized_bytes(path, mode)
        object_type = "SYMLINK"
    else:
        if not path.is_file():
            raise RuntimeError(f"Tracked file not materialized: {rel}")
        data = materialized_bytes(path, mode)
        object_type = "FILE"

    actual_sha = sha256_bytes(data)
    size = len(data)
    pointer = parse_lfs_pointer(data)

    lfs_tracked = rel in lfs_entries or pointer is not None
    lfs_oid = lfs_entries.get(rel)
    if pointer and pointer.get("oid_sha256"):
        lfs_oid = pointer["oid_sha256"]

    if pointer is not None:
        lfs_pointer_count += 1

    if lfs_tracked and pointer is None and lfs_oid is not None:
        # Smudged/materialized LFS object SHA256 should equal the LFS OID.
        if actual_sha != lfs_oid:
            raise RuntimeError(
                f"Materialized LFS content hash does not match LFS OID: {rel}"
            )

    entry = {
        **item,
        "object_type": object_type,
        "materialized": True,
        "size_bytes": size,
        "sha256_materialized": actual_sha,
        "lfs_tracked": lfs_tracked,
        "lfs_oid_sha256": lfs_oid,
        "lfs_pointer_materialized": pointer is not None,
        "lfs_object_size_bytes_from_pointer": (
            pointer.get("object_size_bytes") if pointer else None
        ),
        "recovery_candidate": repo_asset_candidate(rel),
    }

    repo_entries.append(entry)
    repo_total_bytes += size

    if entry["recovery_candidate"]:
        repo_asset_candidates.append(
            {
                "path": rel,
                "size_bytes": size,
                "sha256_materialized": actual_sha,
                "lfs_tracked": lfs_tracked,
                "lfs_oid_sha256": lfs_oid,
                "classification": "CANDIDATE_PATH_ONLY_NOT_ASSERTED_REQUIRED",
            }
        )

print("Tracked entries              :", len(repo_entries))
print("Materialized tracked bytes   :", repo_total_bytes)
print("Repository recovery candidates:", len(repo_asset_candidates))
print("LFS pointer files materialized:", lfs_pointer_count)

parent_inventory_obj = {
    "schema": "stage26_8d6_parent_repository_inventory_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "inventory_commit": EXPECTED_PARENT,
    "repository_url": REPO_URL,
    "branch": BRANCH,
    "inventory_scope": (
        "ALL_TRACKED_INDEX_ENTRIES_AT_EXACT_CLEAN_PARENT_WORKTREE; "
        "8D6 OUTPUTS ARE EXCLUDED BY CONSTRUCTION"
    ),
    "tracked_entry_count": len(repo_entries),
    "total_materialized_tracked_bytes": repo_total_bytes,
    "git_lfs": {
        "patterns_declared": lfs_patterns_declared,
        "git_lfs_available_current_session": git_lfs_available,
        "tracked_path_count": len(lfs_entries),
        "materialized_pointer_count": lfs_pointer_count,
        "fresh_clone_policy": (
            "RUN_GIT_LFS_PULL_IF_LFS_TRACKED_PATH_COUNT_GT_0"
        ),
    },
    "recovery_candidate_policy": (
        "CANDIDATE PATHS ARE DISCOVERY AIDS ONLY; THE FRESH GPU SESSION "
        "MUST USE THE FROZEN REPOSITORY IMPLEMENTATION/REGISTRY TO DETERMINE "
        "ACTUAL REQUIRED INPUTS AND VERIFY THEM BEFORE PROFILING."
    ),
    "recovery_candidates": repo_asset_candidates,
    "entries": repo_entries,
}


# =============================================================================
# 5. RESET-SENSITIVE RUNTIME RECOVERY INVENTORY
# =============================================================================

banner("STAGE26-8D6 :: RESET-SENSITIVE RUNTIME RECOVERY INVENTORY")

runtime_candidates = []
runtime_total_file_count = 0
runtime_total_bytes_metadata = 0
raw_packet_files_excluded = []
candidate_bytes = 0

if RUNTIME_ROOT.exists():
    for path in sorted(RUNTIME_ROOT.rglob("*")):
        if not path.is_file():
            continue

        runtime_total_file_count += 1

        try:
            size = path.stat().st_size
        except OSError:
            continue

        runtime_total_bytes_metadata += size
        suffix = path.suffix.lower()

        if suffix in RAW_PACKET_SUFFIXES:
            raw_packet_files_excluded.append(
                {
                    "path": str(path),
                    "size_bytes": size,
                    "reason": "RAW_PACKET_FILE_EXCLUDED_FROM_HASHING_BY_CPU_CLOSURE_POLICY",
                }
            )
            continue

        if not runtime_asset_candidate(path, size):
            continue

        actual_sha = sha256_file(path)
        candidate_bytes += size
        structure = optional_structure(path)

        runtime_candidates.append(
            {
                "current_session_path": str(path),
                "path_relative_to_runtime_root": str(path.relative_to(RUNTIME_ROOT)),
                "size_bytes": size,
                "sha256": actual_sha,
                "suffix": suffix,
                "structure": structure,
                "classification": "RESET_RECOVERY_CANDIDATE_NOT_ASSERTED_REQUIRED",
            }
        )

runtime_inventory_obj = {
    "schema": "stage26_8d6_runtime_recovery_inventory_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "scientific_parent": EXPECTED_PARENT,
    "runtime_root_current_session": str(RUNTIME_ROOT),
    "runtime_root_exists": RUNTIME_ROOT.exists(),
    "all_runtime_files_metadata_count": runtime_total_file_count,
    "all_runtime_files_metadata_bytes": runtime_total_bytes_metadata,
    "hashed_recovery_candidate_count": len(runtime_candidates),
    "hashed_recovery_candidate_bytes": candidate_bytes,
    "raw_packet_file_count_excluded_from_hashing": len(raw_packet_files_excluded),
    "raw_packet_files_excluded_from_hashing": raw_packet_files_excluded,
    "raw_packet_policy": (
        "RAW PCAP/PCAPNG/CAP FILES ARE NOT HASHED OR USED TO RECREATE THE "
        "COMPACT CORPUS IN THIS CLOSURE. THE GPU PHASE MUST RESTORE THE "
        "ALREADY-ESTABLISHED COMPACT CORPUS/DERIVED ARTIFACTS."
    ),
    "candidate_policy": (
        "THESE ARE RESET-RECOVERY CANDIDATES DISCOVERED FROM THE CURRENT "
        "STAGE26 RUNTIME ROOT. THEY ARE NOT A POST-HOC REQUIRED-ASSET LIST. "
        "THE FRESH GPU BOOTSTRAP MUST RESOLVE ACTUAL REQUIRED ASSETS FROM "
        "THE FROZEN REPOSITORY IMPLEMENTATION AND THEN VERIFY RECOVERED "
        "PATHS/HASHES AGAINST THIS INVENTORY WHEN APPLICABLE."
    ),
    "candidates": runtime_candidates,
}

print("Runtime root exists             :", RUNTIME_ROOT.exists())
print("Runtime files (metadata only)   :", runtime_total_file_count)
print("Runtime recovery candidates     :", len(runtime_candidates))
print("Runtime candidate bytes hashed  :", candidate_bytes)
print("Raw packet files excluded       :", len(raw_packet_files_excluded))


# =============================================================================
# 6. FREEZE FRESH-SESSION GPU BOOTSTRAP CONTRACT
# =============================================================================

banner("STAGE26-8D6 :: FREEZE FRESH-SESSION GPU BOOTSTRAP CONTRACT")

gpu_contract_obj = {
    "schema": "stage26_8d6_gpu_bootstrap_contract_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),

    "cpu_closure_parent": EXPECTED_PARENT,

    "repository": {
        "url": REPO_URL,
        "branch": BRANCH,
        "fresh_session_checkout_policy": (
            "CLONE_REPOSITORY_THEN_CHECKOUT_EXACT_STAGE26_8D6_COMMIT_SHA_"
            "EMITTED_BY_THIS_STAGE"
        ),
        "parent_repository_inventory_file": str(
            PARENT_INVENTORY.relative_to(REPO)
        ),
        "runtime_recovery_inventory_file": str(
            RUNTIME_INVENTORY.relative_to(REPO)
        ),
        "git_lfs_policy": (
            "IF THE FROZEN INVENTORY REPORTS LFS TRACKED PATHS, INSTALL/VERIFY "
            "GIT-LFS AND RUN GIT LFS PULL BEFORE ASSET VERIFICATION."
        ),
    },

    "frozen_identities": {
        "measurement_protocol_sha256": PROTOCOL_SHA,
        "cpu_plan_sha256": CPU_PLAN_SHA,
        "preflight_receipt_sha256": PREFLIGHT_RECEIPT_SHA,
        "effective_cpu_publication_schema_sha256": SCHEMA_SHA,
        "stage26_8d5_audit_sha256": STAGE26_8D5_AUDIT_SHA,
        "stage26_8d5_receipt_sha256": STAGE26_8D5_RECEIPT_SHA,
        "stage26_8d5_manifest_sha256": STAGE26_8D5_MANIFEST_SHA,
    },

    "fresh_kaggle_gpu_bootstrap_sequence": [
        "VERIFY_GPU_VISIBLE_AND_RECORD_GPU_MODEL_UUID_DRIVER_CUDA",
        "READ_GITHUB_TOKEN_FROM_KAGGLE_SECRET_WITHOUT_PRINTING_IT",
        "CLONE_REPOSITORY",
        "CHECKOUT_EXACT_STAGE26_8D6_FINAL_CPU_CLOSURE_COMMIT",
        "VERIFY_HEAD_AND_ORIGIN_MAIN_EXPECTED_COMMIT",
        "VERIFY_REPOSITORY_CLEAN",
        "VERIFY_STAGE26_8D6_CLOSURE_FILES_AND_PARENT_REPOSITORY_INVENTORY",
        "RESTORE_GIT_LFS_OBJECTS_IF_FROZEN_INVENTORY_REQUIRES_LFS",
        "DISCOVER_ACTUAL_REQUIRED_CORPUS_MODELS_PREPROCESSORS_FROM_FROZEN_REPOSITORY_IMPLEMENTATION",
        "RESTORE_ALREADY_ESTABLISHED_COMPACT_CORPUS_AND_REQUIRED_ARTIFACTS",
        "DO_NOT_RECREATE_COMPACT_CORPUS_FROM_MONDAY_OR_ANY_RAW_PCAP",
        "VERIFY_RECOVERED_ASSET_SHA256_SIZE_AND_AVAILABLE_STRUCTURE",
        "VERIFY_PROTOCOL_SCHEMA_TARGET_BATCH_AND_RANDOMIZATION_IDENTITIES",
        "RECORD_FRAMEWORK_AND_GPU_BACKEND_VERSIONS",
        "ALLOW_GPU_PROFILING_ONLY_AFTER_ALL_BOOTSTRAP_GATES_PASS",
    ],

    "gpu_profiling": {
        "target_ids": TARGETS,
        "batch_sizes": BATCHES,
        "warmup_and_timed_runs": WARMUP_TIMED,
        "randomization_seed": SEED,
        "mandatory_gpu_synchronization": (
            "SYNCHRONIZE_GPU_BEFORE_AND_AFTER_EVERY_TIMED_REGION"
        ),
        "gpu_memory_profiling_required": True,
        "supported_backend_policy": (
            "PROFILE_ONLY NATIVE/SUPPORTED GPU EXECUTION PATHS FOR THE "
            "FROZEN MODEL; DO NOT FORCE AN UNSUPPORTED BACKEND."
        ),
        "unsupported_backend_status": "BACKEND_UNAVAILABLE",
        "resource_limit_policy": (
            "PRESERVE GPU OOM/TIMEOUT/BACKEND_UNAVAILABLE AS OBSERVED "
            "RESOURCE/BACKEND OUTCOMES; DO NOT IMPUTE MISSING COST."
        ),
        "model_policy": {
            "retraining_allowed": False,
            "model_conversion_allowed": False,
            "forced_backend_substitution_allowed": False,
            "frozen_predictive_science_changes_allowed": False,
        },
    },

    "scientific_boundaries_carried_forward": {
        "complete_e2e_measurement": "UNAVAILABLE",
        "cross_group_pareto": "PROHIBITED",
        "pr_auc_bootstrap_ci": "PROHIBITED",
        "historical_stage26_4c3": "RETAINED",
        "stage26_8_sensitivity": "DESCRIPTIVE_SEPARATE_NOT_REPLACEMENT",
        "missing_deployment_cost_imputation": "PROHIBITED",
        "post_hoc_best_batch_selection": "PROHIBITED",
    },

    "reset_assumption": {
        "kaggle_accelerator_change_may_reset_session": True,
        "current_kaggle_working_persistence_assumed": False,
        "fresh_session_must_be_self_bootstrapping": True,
    },
}

print("Targets              :", len(TARGETS))
print("Batches              :", BATCHES)
print("Seed                 :", SEED)
print("GPU sync             : MANDATORY")
print("GPU memory profiling : REQUIRED")
print("Unsupported backend  : BACKEND_UNAVAILABLE")
print("Retraining           : PROHIBITED")
print("Model conversion     : PROHIBITED")
print("Raw PCAP recreation  : PROHIBITED")


# =============================================================================
# 7. WRITE PRE-COMMIT CLOSURE FILES
# =============================================================================

banner("STAGE26-8D6 :: WRITE FINAL CPU CLOSURE FILES")

OUT.mkdir(parents=True, exist_ok=False)

atomic_json(PARENT_INVENTORY, parent_inventory_obj)
atomic_json(RUNTIME_INVENTORY, runtime_inventory_obj)
atomic_json(GPU_CONTRACT, gpu_contract_obj)

now = datetime.now(timezone.utc).isoformat()

closure_receipt_obj = {
    "schema": "stage26_8d6_final_cpu_closure_receipt_v1",
    "created_at_utc": now,
    "scientific_parent": EXPECTED_PARENT,

    "cpu_publication_artifact_audit": {
        "commit": EXPECTED_PARENT,
        "verdict": "CPU_PUBLICATION_ARTIFACT_PACKAGE_PROTOCOL_CONFORMANT",
        "audit_sha256": STAGE26_8D5_AUDIT_SHA,
        "receipt_sha256": STAGE26_8D5_RECEIPT_SHA,
        "manifest_sha256": STAGE26_8D5_MANIFEST_SHA,
    },

    "frozen_identities": {
        "measurement_protocol_sha256": PROTOCOL_SHA,
        "cpu_plan_sha256": CPU_PLAN_SHA,
        "preflight_receipt_sha256": PREFLIGHT_RECEIPT_SHA,
        "effective_schema_sha256": SCHEMA_SHA,
    },

    "repository_recovery": {
        "repository_url": REPO_URL,
        "branch": BRANCH,
        "parent_inventory_sha256": sha256_file(PARENT_INVENTORY),
        "runtime_recovery_inventory_sha256": sha256_file(RUNTIME_INVENTORY),
        "gpu_bootstrap_contract_sha256": sha256_file(GPU_CONTRACT),
        "exact_final_commit_policy": (
            "THE FRESH GPU SESSION MUST CHECK OUT THE FINAL 8D6 COMMIT SHA "
            "EMITTED AFTER THIS RECEIPT IS COMMITTED."
        ),
    },

    "cpu_phase": "COMPLETE_AFTER_8D6_COMMIT_PUSH_REMOTE_VERIFICATION",
    "gpu_used_in_stage26_8d6": False,
    "gpu_allowed_after_8d6_commit_push_remote_verification": True,

    "scientific_state": {
        "cpu_measurement_science": "FROZEN",
        "cpu_publication_tables": "FROZEN_AND_AUDITED",
        "cpu_publication_figures": "FROZEN_AND_AUDITED",
        "complete_e2e": "UNAVAILABLE",
        "historical_stage26_4c3": "RETAINED",
        "stage26_8_sensitivity": "DESCRIPTIVE_SEPARATE_NOT_REPLACEMENT",
    },

    "non_computation": {
        "timing_executed": False,
        "inference_executed": False,
        "model_retraining_executed": False,
        "model_conversion_executed": False,
        "bootstrap_executed": False,
        "raw_pcap_hashed_or_read_for_science": False,
        "gpu_used": False,
    },

    "next_phase": "FRESH_KAGGLE_GPU_BOOTSTRAP",
}

atomic_json(CLOSURE_RECEIPT, closure_receipt_obj)

closure_files_before_manifest = [
    PARENT_INVENTORY,
    RUNTIME_INVENTORY,
    GPU_CONTRACT,
    CLOSURE_RECEIPT,
]

manifest_obj = {
    "schema": "stage26_8d6_final_cpu_closure_manifest_v1",
    "created_at_utc": now,
    "scientific_parent": EXPECTED_PARENT,
    "files": [
        {
            "path": str(path.relative_to(REPO)),
            "sha256": sha256_file(path),
            "size_bytes": path.stat().st_size,
        }
        for path in closure_files_before_manifest
    ],
}

atomic_json(CLOSURE_MANIFEST, manifest_obj)

print("Parent inventory SHA256 :", sha256_file(PARENT_INVENTORY))
print("Runtime inventory SHA256:", sha256_file(RUNTIME_INVENTORY))
print("GPU contract SHA256     :", sha256_file(GPU_CONTRACT))
print("Closure receipt SHA256  :", sha256_file(CLOSURE_RECEIPT))
print("Closure manifest SHA256 :", sha256_file(CLOSURE_MANIFEST))


# =============================================================================
# 8. PRE-COMMIT GIT AUDIT
# =============================================================================

banner("STAGE26-8D6 :: PRE-COMMIT GIT AUDIT")

lines = [
    line
    for line in git("status", "--porcelain").splitlines()
    if line.strip()
]

for line in lines:
    print(line)

prefix = "?? " + str(OUT.relative_to(REPO))

if not lines or any(not line.startswith(prefix) for line in lines):
    raise RuntimeError(
        "Unexpected pre-commit repository state."
    )


# =============================================================================
# 9. COMMIT
# =============================================================================

banner("STAGE26-8D6 :: COMMIT")

git("add", str(OUT.relative_to(REPO)))

staged = git("diff", "--cached", "--name-only").splitlines()

expected_staged = sorted(
    str(path.relative_to(REPO))
    for path in [
        PARENT_INVENTORY,
        RUNTIME_INVENTORY,
        GPU_CONTRACT,
        CLOSURE_RECEIPT,
        CLOSURE_MANIFEST,
    ]
)

print("Staged files:")
for path in staged:
    print(" ", path)

if sorted(staged) != expected_staged:
    raise RuntimeError("Unexpected staged file set.")

git("commit", "-m", COMMIT_MSG)

new_head = git("rev-parse", "HEAD")
parent = git("rev-parse", "HEAD^")
subject = git("log", "-1", "--pretty=%s")

print("Parent :", parent)
print("HEAD   :", new_head)
print("Subject:", subject)

if parent != EXPECTED_PARENT:
    raise RuntimeError("8D6 parent changed.")
if subject != COMMIT_MSG:
    raise RuntimeError("8D6 commit subject changed.")


# =============================================================================
# 10. PUSH USING KAGGLE SECRET
# =============================================================================

banner("STAGE26-8D6 :: PUSH")

from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")

if not token or len(token.strip()) < 20:
    raise RuntimeError("GITHUB_TOKEN unavailable.")

fd, askpass_name = tempfile.mkstemp(
    prefix="stage26_8d6_askpass_",
    suffix=".sh",
)
os.close(fd)
askpass = Path(askpass_name)

try:
    askpass.write_text(
        "#!/bin/sh\n"
        "case \"$1\" in\n"
        "  *Username*) printf \"%s\\n\" \"x-access-token\" ;;\n"
        "  *Password*) printf \"%s\\n\" \"$STAGE26_GITHUB_TOKEN\" ;;\n"
        "  *) printf \"%s\\n\" \"\" ;;\n"
        "esac\n",
        encoding="utf-8",
    )

    askpass.chmod(
        askpass.stat().st_mode
        | stat.S_IXUSR
        | stat.S_IXGRP
        | stat.S_IXOTH
    )

    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["STAGE26_GITHUB_TOKEN"] = token.strip()

    push_out = git("push", "origin", "main", env=env)
    print(push_out)

finally:
    askpass.unlink(missing_ok=True)
    token = None
    if "env" in locals():
        env.pop("STAGE26_GITHUB_TOKEN", None)


# =============================================================================
# 11. REMOTE BYTE VERIFICATION
# =============================================================================

banner("STAGE26-8D6 :: REMOTE BYTE VERIFICATION")

git("fetch", "origin", "main")

local_head = git("rev-parse", "HEAD")
remote_head = git("rev-parse", "origin/main")

print("Local HEAD :", local_head)
print("origin/main:", remote_head)

if local_head != new_head or remote_head != new_head:
    raise RuntimeError("Local/remote final CPU commit mismatch.")

closure_paths = [
    PARENT_INVENTORY,
    RUNTIME_INVENTORY,
    GPU_CONTRACT,
    CLOSURE_RECEIPT,
    CLOSURE_MANIFEST,
]

for path in closure_paths:
    rel = str(path.relative_to(REPO))
    local_bytes = path.read_bytes()
    remote_bytes = git_blob("origin/main", rel)

    local_sha = sha256_bytes(local_bytes)
    remote_sha = sha256_bytes(remote_bytes)
    ok = local_bytes == remote_bytes

    print(f"{'PASS' if ok else 'FAIL'} {rel}")
    print("  local :", local_sha)
    print("  remote:", remote_sha)

    if not ok:
        raise RuntimeError(f"Remote byte mismatch: {rel}")


# =============================================================================
# 12. FINAL CLEAN GATE + GPU AUTHORIZATION
# =============================================================================

final_status = git("status", "--porcelain")

if final_status:
    raise RuntimeError(
        "Repository not clean after Stage26-8D6."
    )

if git("rev-parse", "HEAD") != git("rev-parse", "origin/main"):
    raise RuntimeError(
        "HEAD/origin diverged after Stage26-8D6."
    )

banner("STAGE26-8D6 COMPLETE :: FINAL CPU CLOSURE")

print("FINAL_CPU_CLOSURE_ANCHOR :", new_head)
print("HEAD == origin/main      :", True)
print("Repo clean               :", True)
print("Remote closure bytes     : VERIFIED")
print()
print("CPU publication package  : PROTOCOL CONFORMANT")
print("CPU measurement science  : FROZEN")
print("CPU_PHASE                 : COMPLETE")
print()
print("GPU_USED_IN_THIS_STAGE    : FALSE")
print("GPU_ALLOWED               : TRUE")
print()
print("KAGGLE_RESET_EXPECTED     : YES")
print("CURRENT_WORKING_PERSISTENCE_ASSUMED: NO")
print()
print("FRESH GPU SESSION MUST:")
print(f"  1. clone {REPO_URL}")
print(f"  2. checkout EXACT commit {new_head}")
print("  3. verify closure + parent repository inventory")
print("  4. run git lfs pull if the frozen inventory requires it")
print("  5. restore the already-established compact corpus/artifacts")
print("  6. NEVER recreate the compact corpus from Monday/raw PCAP")
print("  7. verify corpus/models/preprocessors by hash/size/structure")
print("  8. record GPU/driver/CUDA/framework identity")
print("  9. begin GPU profiling only after every bootstrap gate passes")
print()
print("NEXT:")
print("  Activate the Kaggle GPU accelerator.")
print("  Expect a fresh runtime.")
print(f"  The GPU bootstrap anchor is {new_head}.")



STAGE26-8D6 :: DURABLE STATE + CPU-ONLY GATE
Expected parent : 9c560a37d7a5d3bd79ad1a72a103e64c57548f26
Local HEAD      : 9c560a37d7a5d3bd79ad1a72a103e64c57548f26
origin/main     : 9c560a37d7a5d3bd79ad1a72a103e64c57548f26
Repo clean      : True
Origin URL      : https://github.com/themubasshir/ids2018-validation-safe-ablation.git
nvidia-smi GPU visible : False
torch.cuda available  : False
torch version          : 2.10.0+cpu

STAGE26-8D6 :: 8D5 / SCHEMA / PROTOCOL IDENTITY
stage26_8d5_cpu_publication_artifact_audit.json
  expected=e4375ab82ce31a9120038950c268397570cf7fb741339a843433417bfa279225
  actual  =e4375ab82ce31a9120038950c268397570cf7fb741339a843433417bfa279225
stage26_8d5_cpu_publication_artifact_audit_receipt.json
  expected=f97f8396b1fa7eaa3315060007a3def4b4266ac3e1ca5dbf5d52bef74394fb42
  actual  =f97f8396b1fa7eaa3315060007a3def4b4266ac3e1ca5dbf5d52bef74394fb42
stage26_8d5_cpu_publication_artifact_audit_manifest.json
  expected=5def9c65b11adddb2243f83455f3531642a278f7ccd44

In [1]:
# ==================================================================================================
# STAGE26-G0A :: FRESH GPU BOOTSTRAP + RESET-RECOVERY DISCOVERY
# --------------------------------------------------------------------------------------------------
# PURPOSE
#   * Start from the exact Stage26-8D6 CPU closure anchor.
#   * Prove that a real CUDA-capable Kaggle GPU runtime is active.
#   * Verify the frozen CPU closure/protocol identities byte-for-byte.
#   * Discover reset-recoverable compact-corpus / representation artifacts from /kaggle/input.
#   * Discover frozen implementation references required by the eight GPU targets.
#
# THIS CELL DOES NOT:
#   * run inference timing
#   * allocate benchmark batches
#   * modify model science
#   * retrain / convert models
#   * recreate anything from raw PCAP
#   * commit or push anything
#
# After this cell, paste the COMPLETE output before running anything else.
# ==================================================================================================

from __future__ import annotations

import os
import sys
import json
import hashlib
import platform
import socket
import subprocess
import textwrap
from pathlib import Path
from datetime import datetime, timezone

# --------------------------------------------------------------------------------------------------
# FROZEN IDENTITIES
# --------------------------------------------------------------------------------------------------

REPO_URL = "https://github.com/themubasshir/ids2018-validation-safe-ablation.git"
REPO_DIR = Path("/kaggle/working/ids2018-validation-safe-ablation")

FINAL_CPU_CLOSURE_ANCHOR = "304e5613627a744cbd5d369857f8ac5667a520eb"

EXPECTED = {
    "stage26_8d6_parent_repository_inventory.json":
        "8498a9690b7559811ef185f5e639698e73a912099cf6fee89e5fd8a4ad9ccbfc",

    "stage26_8d6_runtime_recovery_inventory.json":
        "5ab7158c7fbdd223545b811b2d8417cd4050b5a47a87a333a5eadac339f0ea5c",

    "stage26_8d6_gpu_bootstrap_contract.json":
        "09dfeb386f462597d4fca76aad5676d20901017c6f8175d221d2f85e108d59b7",

    "stage26_8d6_final_cpu_closure_receipt.json":
        "6d946d710f4f5e5a47b13fce927eee5d36203acbbb7fd0e0e0237cc6133160fa",

    "stage26_8d6_final_cpu_closure_manifest.json":
        "ca66254c3075c0aa27285ad8fe4ae062275875fc48775be6d8af6e3f65e1455c",
}

MEASUREMENT_PROTOCOL_SHA256 = (
    "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
)

CPU_PLAN_SHA256 = (
    "b66b537f9c3652fee81aeeec956b7c825938967db67d32ccbfd09bd49bf67363"
)

EFFECTIVE_SCHEMA_SHA256 = (
    "973cc58ed6572d30b1cce94f3659226d99b185390c2aca92d9ed030e50358ad0"
)

PREFLIGHT_RECEIPT_SHA256 = (
    "4c7fc55a0e44e53587d385989dfec4f3f57f20768c7dd4a2d08e2936d2ba21e5"
)

FROZEN_TARGETS = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
    "ENS_LGBM_XGB_EQUAL",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
]

FROZEN_BATCHES = [1, 64, 256, 1024, 8192]

FROZEN_RUNS = {
    1:    {"warmup": 50, "timed": 200},
    64:   {"warmup": 30, "timed": 150},
    256:  {"warmup": 20, "timed": 100},
    1024: {"warmup": 10, "timed": 50},
    8192: {"warmup": 5, "timed": 20},
}

RANDOMIZATION_SEED = 26042

CLOSURE_DIR_REL = Path(
    "results/stage26_deployment_profiling/"
    "stage26_8d6_final_cpu_closure"
)

# --------------------------------------------------------------------------------------------------
# UTILITIES
# --------------------------------------------------------------------------------------------------

def banner(title: str) -> None:
    print()
    print("=" * 120)
    print(title)
    print("=" * 120)


def run(
    cmd,
    *,
    cwd: Path | None = None,
    check: bool = True,
    capture: bool = True,
    env=None,
):
    if isinstance(cmd, str):
        args = cmd
        shell = True
    else:
        args = [str(x) for x in cmd]
        shell = False

    proc = subprocess.run(
        args,
        cwd=str(cwd) if cwd else None,
        shell=shell,
        check=False,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.PIPE if capture else None,
        env=env,
    )

    if check and proc.returncode != 0:
        print("\nCOMMAND FAILED")
        print("cmd:", cmd)
        print("returncode:", proc.returncode)
        if capture:
            print("stdout:")
            print(proc.stdout)
            print("stderr:")
            print(proc.stderr)
        raise RuntimeError(f"Command failed: {cmd}")

    return proc


def sha256_file(path: Path, chunk_size: int = 16 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def git(args, check=True):
    return run(["git", *args], cwd=REPO_DIR, check=check)


def human_bytes(n: int) -> str:
    n = int(n)
    gib = n / (1024 ** 3)
    mib = n / (1024 ** 2)
    if gib >= 1:
        return f"{gib:.3f} GiB"
    return f"{mib:.3f} MiB"


def json_load(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


# --------------------------------------------------------------------------------------------------
# A. FRESH RUNTIME IDENTITY
# --------------------------------------------------------------------------------------------------

banner("STAGE26-G0A :: FRESH RUNTIME IDENTITY")

print("timestamp_utc :", datetime.now(timezone.utc).isoformat())
print("hostname      :", socket.gethostname())
print("python        :", sys.version.replace("\n", " "))
print("executable    :", sys.executable)
print("platform      :", platform.platform())
print("machine       :", platform.machine())
print("working dir   :", os.getcwd())
print("/kaggle/input :", Path("/kaggle/input").exists())
print("/kaggle/working:", Path("/kaggle/working").exists())

# --------------------------------------------------------------------------------------------------
# B. GPU / CUDA HARD GATE
# --------------------------------------------------------------------------------------------------

banner("STAGE26-G0A :: GPU / CUDA HARD GATE")

nvsmi = run(["nvidia-smi"], check=False)

print("nvidia-smi return code:", nvsmi.returncode)

if nvsmi.returncode != 0:
    print(nvsmi.stdout)
    print(nvsmi.stderr)
    raise RuntimeError(
        "NVIDIA GPU is not visible. "
        "Do not continue Stage26 GPU profiling."
    )

query = run(
    [
        "nvidia-smi",
        "--query-gpu=name,uuid,driver_version,memory.total",
        "--format=csv,noheader,nounits",
    ],
    check=True,
)

gpu_rows = [x.strip() for x in query.stdout.splitlines() if x.strip()]

print("GPU count:", len(gpu_rows))

for i, row in enumerate(gpu_rows):
    print(f"GPU[{i}] :", row)

if len(gpu_rows) < 1:
    raise RuntimeError("nvidia-smi succeeded but returned no GPUs.")

cuda_version_line = None

for line in nvsmi.stdout.splitlines():
    if "CUDA Version:" in line:
        cuda_version_line = line.strip()
        break

print("nvidia-smi CUDA line:", cuda_version_line)

# --------------------------------------------------------------------------------------------------
# C. PYTHON ML STACK / CUDA BACKEND
# --------------------------------------------------------------------------------------------------

banner("STAGE26-G0A :: FRAMEWORK / BACKEND IDENTITY")

versions = {}

def get_version(name):
    try:
        mod = __import__(name)
        return getattr(mod, "__version__", "UNKNOWN")
    except Exception as e:
        return f"IMPORT_ERROR:{type(e).__name__}:{e}"


for package in [
    "numpy",
    "pandas",
    "sklearn",
    "xgboost",
    "lightgbm",
    "catboost",
    "torch",
]:
    versions[package] = get_version(package)
    print(f"{package:10s}: {versions[package]}")

try:
    import torch

    print()
    print("torch.cuda.is_available :", torch.cuda.is_available())
    print("torch.version.cuda      :", torch.version.cuda)
    print("torch.backends.cudnn    :", torch.backends.cudnn.version())

    if not torch.cuda.is_available():
        raise RuntimeError(
            "NVIDIA GPU is visible but PyTorch CUDA is unavailable. "
            "The frozen FT/CNN/ViT GPU targets cannot proceed."
        )

    print("torch CUDA device count :", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(
            f"torch device[{i}]       : "
            f"{torch.cuda.get_device_name(i)} | "
            f"CC {props.major}.{props.minor} | "
            f"{props.total_memory / (1024 ** 3):.3f} GiB"
        )

except Exception:
    raise

# --------------------------------------------------------------------------------------------------
# D. KAGGLE GITHUB SECRET PRESENCE — NEVER PRINT SECRET
# --------------------------------------------------------------------------------------------------

banner("STAGE26-G0A :: KAGGLE SECRET DISCOVERY")

github_secret_label = None

try:
    from kaggle_secrets import UserSecretsClient
    usc = UserSecretsClient()

    for label in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:
        try:
            value = usc.get_secret(label)
            if value:
                github_secret_label = label
                break
        except Exception:
            pass

except Exception as e:
    print("Kaggle secret client error:", repr(e))

print("Usable GitHub secret label:", github_secret_label or "NOT_FOUND")

if github_secret_label is None:
    raise RuntimeError(
        "No usable GitHub token secret was found. "
        "Stage26 GPU results must later be committed/pushed durably."
    )

# --------------------------------------------------------------------------------------------------
# E. CLONE / CHECKOUT EXACT FINAL CPU CLOSURE
# --------------------------------------------------------------------------------------------------

banner("STAGE26-G0A :: EXACT REPOSITORY RESTORE")

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git repository."
        )

    status_before = git(["status", "--porcelain"]).stdout.strip()

    if status_before:
        print(status_before)
        raise RuntimeError(
            "Existing repository is dirty. "
            "Refusing to destroy or overwrite fresh-session state."
        )

    current_origin = git(["remote", "get-url", "origin"]).stdout.strip()

    if current_origin.rstrip("/") != REPO_URL.rstrip("/"):
        raise RuntimeError(
            f"Unexpected origin URL: {current_origin}"
        )

    print("Existing clean clone found.")
    git(["fetch", "--prune", "origin"])

else:
    print("Fresh clone required.")
    run(
        [
            "git",
            "clone",
            REPO_URL,
            str(REPO_DIR),
        ],
        cwd=Path("/kaggle/working"),
    )

git(["fetch", "--prune", "origin"])

# Exact commit must exist.
probe = git(
    ["cat-file", "-e", f"{FINAL_CPU_CLOSURE_ANCHOR}^{{commit}}"],
    check=False,
)

if probe.returncode != 0:
    raise RuntimeError(
        f"Frozen CPU closure commit does not exist locally: "
        f"{FINAL_CPU_CLOSURE_ANCHOR}"
    )

# Recreate local main exactly at the frozen handoff anchor.
git(["checkout", "-B", "main", FINAL_CPU_CLOSURE_ANCHOR])

local_head = git(["rev-parse", "HEAD"]).stdout.strip()
origin_main = git(["rev-parse", "origin/main"]).stdout.strip()
repo_clean = git(["status", "--porcelain"]).stdout.strip() == ""
origin_url = git(["remote", "get-url", "origin"]).stdout.strip()

print("Expected anchor :", FINAL_CPU_CLOSURE_ANCHOR)
print("Local HEAD      :", local_head)
print("origin/main     :", origin_main)
print("Repo clean      :", repo_clean)
print("Origin URL      :", origin_url)

if local_head != FINAL_CPU_CLOSURE_ANCHOR:
    raise RuntimeError("Local HEAD does not equal frozen CPU closure anchor.")

if origin_main != FINAL_CPU_CLOSURE_ANCHOR:
    raise RuntimeError(
        "origin/main no longer equals the frozen CPU closure anchor. "
        "Do not silently advance the scientific parent."
    )

if not repo_clean:
    raise RuntimeError("Repository is not clean after exact checkout.")

# --------------------------------------------------------------------------------------------------
# F. VERIFY 8D6 CLOSURE BYTES
# --------------------------------------------------------------------------------------------------

banner("STAGE26-G0A :: FINAL CPU CLOSURE BYTE VERIFICATION")

closure_dir = REPO_DIR / CLOSURE_DIR_REL

if not closure_dir.exists():
    raise RuntimeError(f"Missing closure directory: {closure_dir}")

for filename, expected_sha in EXPECTED.items():
    p = closure_dir / filename

    if not p.exists():
        raise RuntimeError(f"Missing frozen closure artifact: {p}")

    actual_sha = sha256_file(p)

    print(filename)
    print("  expected:", expected_sha)
    print("  actual  :", actual_sha)

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"Frozen closure byte mismatch: {filename}"
        )

contract_path = closure_dir / "stage26_8d6_gpu_bootstrap_contract.json"
runtime_inventory_path = closure_dir / "stage26_8d6_runtime_recovery_inventory.json"
parent_inventory_path = closure_dir / "stage26_8d6_parent_repository_inventory.json"

contract = json_load(contract_path)
runtime_inventory = json_load(runtime_inventory_path)
parent_inventory = json_load(parent_inventory_path)

# --------------------------------------------------------------------------------------------------
# G. VERIFY FROZEN PROTOCOL CONTRACT
# --------------------------------------------------------------------------------------------------

banner("STAGE26-G0A :: FROZEN GPU CONTRACT VERIFICATION")

frozen_ids = contract["frozen_identities"]
gpu_contract = contract["gpu_profiling"]

checks = {
    "measurement_protocol_sha256":
        frozen_ids["measurement_protocol_sha256"] == MEASUREMENT_PROTOCOL_SHA256,

    "cpu_plan_sha256":
        frozen_ids["cpu_plan_sha256"] == CPU_PLAN_SHA256,

    "effective_cpu_publication_schema_sha256":
        frozen_ids["effective_cpu_publication_schema_sha256"] == EFFECTIVE_SCHEMA_SHA256,

    "preflight_receipt_sha256":
        frozen_ids["preflight_receipt_sha256"] == PREFLIGHT_RECEIPT_SHA256,

    "targets":
        gpu_contract["target_ids"] == FROZEN_TARGETS,

    "batches":
        gpu_contract["batch_sizes"] == FROZEN_BATCHES,

    "seed":
        int(gpu_contract["randomization_seed"]) == RANDOMIZATION_SEED,

    "gpu_sync":
        gpu_contract["mandatory_gpu_synchronization"]
        == "SYNCHRONIZE_GPU_BEFORE_AND_AFTER_EVERY_TIMED_REGION",

    "gpu_memory":
        bool(gpu_contract["gpu_memory_profiling_required"]) is True,

    "unsupported_backend":
        gpu_contract["unsupported_backend_status"] == "BACKEND_UNAVAILABLE",

    "retraining_prohibited":
        gpu_contract["model_policy"]["retraining_allowed"] is False,

    "conversion_prohibited":
        gpu_contract["model_policy"]["model_conversion_allowed"] is False,

    "backend_substitution_prohibited":
        gpu_contract["model_policy"]["forced_backend_substitution_allowed"] is False,
}

contract_runs = {
    int(k): {
        "warmup": int(v["warmup"]),
        "timed": int(v["timed"]),
    }
    for k, v in gpu_contract["warmup_and_timed_runs"].items()
}

checks["warmup_timed_runs"] = contract_runs == FROZEN_RUNS

for name, ok in checks.items():
    print(f"{name:42s}: {ok}")

if not all(checks.values()):
    bad = [k for k, v in checks.items() if not v]
    raise RuntimeError(
        "Frozen GPU bootstrap contract mismatch: " + ", ".join(bad)
    )

# --------------------------------------------------------------------------------------------------
# H. GIT LFS GATE
# --------------------------------------------------------------------------------------------------

banner("STAGE26-G0A :: GIT LFS GATE")

lfs_available = run(["git", "lfs", "version"], check=False).returncode == 0

if lfs_available:
    lfs_listing = git(["lfs", "ls-files"], check=False).stdout.strip()
else:
    lfs_listing = ""

lfs_paths = [
    line for line in lfs_listing.splitlines()
    if line.strip()
]

print("git-lfs available    :", lfs_available)
print("git-lfs tracked paths:", len(lfs_paths))

if lfs_paths:
    print("Frozen repository uses LFS; restoring objects...")
    git(["lfs", "pull"])
else:
    print("No Git-LFS object restoration required.")

# --------------------------------------------------------------------------------------------------
# I. SCAN KAGGLE INPUT — METADATA ONLY FIRST
# --------------------------------------------------------------------------------------------------

banner("STAGE26-G0A :: KAGGLE INPUT INVENTORY")

input_root = Path("/kaggle/input")

input_files = []

if input_root.exists():
    for p in sorted(input_root.rglob("*")):
        if p.is_file():
            try:
                st = p.stat()
                input_files.append(
                    {
                        "path": str(p),
                        "name": p.name,
                        "size_bytes": int(st.st_size),
                        "suffix": p.suffix.lower(),
                    }
                )
            except OSError:
                pass

print("Input files discovered:", len(input_files))
print(
    "Total input bytes      :",
    human_bytes(sum(x["size_bytes"] for x in input_files)),
)

# Print only top 40 largest files, metadata only.
print()
print("Largest /kaggle/input files:")
for row in sorted(
    input_files,
    key=lambda x: x["size_bytes"],
    reverse=True
)[:40]:
    print(
        f"  {human_bytes(row['size_bytes']):>12s}  "
        f"{row['path']}"
    )

# --------------------------------------------------------------------------------------------------
# J. RECOVERY-CANDIDATE MATCHING AGAINST FROZEN 8D6 INVENTORY
# --------------------------------------------------------------------------------------------------

banner("STAGE26-G0A :: RESET-RECOVERY CANDIDATE DISCOVERY")

runtime_candidates = runtime_inventory.get("candidates", [])

print("Frozen runtime recovery candidates:", len(runtime_candidates))

# PCAP-like files are never hashed here.
RAW_PACKET_SUFFIXES = {
    ".pcap",
    ".pcapng",
    ".cap",
}

matches = []
missing = []

for candidate in runtime_candidates:
    basename = Path(candidate["current_session_path"]).name
    expected_size = int(candidate["size_bytes"])
    expected_sha = candidate["sha256"]

    possible = [
        Path(x["path"])
        for x in input_files
        if x["name"] == basename
        and int(x["size_bytes"]) == expected_size
        and x["suffix"] not in RAW_PACKET_SUFFIXES
    ]

    print()
    print("Candidate :", candidate["path_relative_to_runtime_root"])
    print("Expected  :", human_bytes(expected_size), expected_sha)

    if not possible:
        print("Input hit : NONE")
        missing.append(candidate["path_relative_to_runtime_root"])
        continue

    verified_this_candidate = []

    for p in possible:
        print("Input hit :", p)
        actual_sha = sha256_file(p)
        ok = actual_sha == expected_sha

        print("SHA256    :", actual_sha)
        print("Verified  :", ok)

        verified_this_candidate.append(
            {
                "path": str(p),
                "sha256": actual_sha,
                "verified": ok,
            }
        )

        if ok:
            matches.append(
                {
                    "candidate": candidate,
                    "input_path": str(p),
                }
            )

    if not any(x["verified"] for x in verified_this_candidate):
        raise RuntimeError(
            f"Found same-name/same-size recovery candidate but SHA256 "
            f"did not match frozen inventory: {basename}"
        )

# --------------------------------------------------------------------------------------------------
# K. EXPLICIT COMPACT-CORPUS GATE
# --------------------------------------------------------------------------------------------------

banner("STAGE26-G0A :: COMPACT CORPUS DISCOVERY")

COMPACT_NAME = "stage20-Monday-compact-corpus-v1.tar"
COMPACT_SIZE = 595_261_440
COMPACT_SHA = "4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20"

compact_hits = [
    Path(x["path"])
    for x in input_files
    if x["name"] == COMPACT_NAME
    and int(x["size_bytes"]) == COMPACT_SIZE
]

print("Expected compact corpus name :", COMPACT_NAME)
print("Expected compact corpus bytes:", COMPACT_SIZE)
print("Expected compact corpus SHA  :", COMPACT_SHA)
print("Matching-size/name hits      :", len(compact_hits))

verified_compact_hits = []

for p in compact_hits:
    print()
    print("Checking:", p)
    actual = sha256_file(p)
    ok = actual == COMPACT_SHA

    print("  actual SHA:", actual)
    print("  verified  :", ok)

    if ok:
        verified_compact_hits.append(str(p))

if not verified_compact_hits:
    print()
    print(
        "WARNING: exact compact-corpus TAR was not found as a directly "
        "attached /kaggle/input file."
    )
    print(
        "This does NOT authorize recreating it from Monday/raw PCAP."
    )
else:
    print()
    print("Verified compact corpus input(s):")
    for p in verified_compact_hits:
        print(" ", p)

# --------------------------------------------------------------------------------------------------
# L. VERIFY ANY DIRECTLY-ATTACHED DERIVED REPRESENTATION FILES
# --------------------------------------------------------------------------------------------------

banner("STAGE26-G0A :: REPRESENTATION-ASSET DISCOVERY")

REP_EXPECTED = {
    "encoded_bytes.bin": {
        "size": 522_845_159,
        "sha": "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",
    },
    "flow_offsets.npy": {
        "size": 4_228_208,
        "sha": "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",
    },
    "packet_lengths.npy": {
        "size": 67_649_280,
        "sha": "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
    },
}

verified_representation = {}

for name, meta in REP_EXPECTED.items():

    hits = [
        Path(x["path"])
        for x in input_files
        if x["name"] == name
        and int(x["size_bytes"]) == meta["size"]
    ]

    print()
    print(name)
    print("  expected size:", meta["size"])
    print("  expected SHA :", meta["sha"])
    print("  hits         :", len(hits))

    good = []

    for p in hits:
        actual = sha256_file(p)
        ok = actual == meta["sha"]

        print("  path         :", p)
        print("  actual SHA   :", actual)
        print("  verified     :", ok)

        if ok:
            good.append(str(p))

    verified_representation[name] = good

# --------------------------------------------------------------------------------------------------
# M. FROZEN IMPLEMENTATION DISCOVERY
# --------------------------------------------------------------------------------------------------

banner("STAGE26-G0A :: FROZEN TARGET IMPLEMENTATION DISCOVERY")

# We do not guess model paths here. We search the frozen repository at the exact
# CPU closure anchor and print the implementation references for the next recovery cell.

implementation_hits = {}

for target in FROZEN_TARGETS:

    proc = git(
        [
            "grep",
            "-n",
            "-I",
            "-F",
            target,
            "--",
            ".",
        ],
        check=False,
    )

    lines = [
        line.strip()
        for line in proc.stdout.splitlines()
        if line.strip()
    ]

    implementation_hits[target] = lines

    print()
    print(target)
    print("-" * len(target))

    if not lines:
        print("  NO REPOSITORY TEXT HIT")
    else:
        print("  repository hits:", len(lines))

        # Keep output useful without flooding the notebook.
        for line in lines[:30]:
            print(" ", line)

        if len(lines) > 30:
            print(f"  ... {len(lines) - 30} additional hit(s) omitted")

# --------------------------------------------------------------------------------------------------
# N. REPOSITORY RECOVERY-CANDIDATE SUMMARY
# --------------------------------------------------------------------------------------------------

banner("STAGE26-G0A :: FROZEN PARENT INVENTORY SUMMARY")

print("Parent inventory schema :", parent_inventory.get("schema"))
print(
    "Inventory top-level keys:",
    sorted(parent_inventory.keys())
)

# Recursively summarize likely file/path records without making semantic assumptions.
def walk_records(obj):
    if isinstance(obj, dict):
        yield obj
        for value in obj.values():
            yield from walk_records(value)
    elif isinstance(obj, list):
        for value in obj:
            yield from walk_records(value)

interesting_records = []

for rec in walk_records(parent_inventory):

    path_value = None

    for key in [
        "path",
        "repo_path",
        "repository_path",
        "tracked_path",
        "path_relative_to_repo",
    ]:
        if key in rec and isinstance(rec[key], str):
            path_value = rec[key]
            break

    if not path_value:
        continue

    low = path_value.lower()

    if any(
        token in low
        for token in [
            "model",
            "checkpoint",
            "preprocess",
            "scaler",
            "encoder",
            "xgboost",
            "lightgbm",
            "catboost",
            "cnn",
            "vit",
            "transformer",
            ".pt",
            ".pth",
            ".ckpt",
            ".json",
            ".joblib",
            ".pkl",
            ".bin",
        ]
    ):
        interesting_records.append(rec)

print(
    "Potential model/preprocessor/recovery records:",
    len(interesting_records)
)

for rec in interesting_records[:100]:

    path_value = (
        rec.get("path")
        or rec.get("repo_path")
        or rec.get("repository_path")
        or rec.get("tracked_path")
        or rec.get("path_relative_to_repo")
    )

    size_value = (
        rec.get("size_bytes")
        or rec.get("materialized_size_bytes")
        or rec.get("bytes")
    )

    sha_value = (
        rec.get("sha256")
        or rec.get("materialized_sha256")
    )

    print()
    print("PATH :", path_value)

    if size_value is not None:
        print("SIZE :", size_value)

    if sha_value is not None:
        print("SHA  :", sha_value)

if len(interesting_records) > 100:
    print()
    print(
        f"... {len(interesting_records) - 100} additional parent-inventory "
        f"records omitted from display."
    )

# --------------------------------------------------------------------------------------------------
# O. FINAL BOOTSTRAP-DISCOVERY STATE
# --------------------------------------------------------------------------------------------------

banner("STAGE26-G0A COMPLETE :: GPU BOOTSTRAP DISCOVERY")

print("FINAL_CPU_CLOSURE_ANCHOR :", FINAL_CPU_CLOSURE_ANCHOR)
print("Local HEAD               :", local_head)
print("origin/main              :", origin_main)
print("Repo clean               :", git(["status", "--porcelain"]).stdout.strip() == "")
print()
print("NVIDIA GPU visible       :", True)
print("torch CUDA available     :", torch.cuda.is_available())
print("GPU count                :", torch.cuda.device_count())
print("GitHub secret available  :", github_secret_label is not None)
print()
print("Frozen targets           :", len(FROZEN_TARGETS))
print("Frozen batches           :", FROZEN_BATCHES)
print("Randomization seed       :", RANDOMIZATION_SEED)
print("GPU synchronization      : MANDATORY")
print("GPU memory profiling     : REQUIRED")
print("Unsupported native path  : BACKEND_UNAVAILABLE")
print()
print("Verified compact TAR hits:", len(verified_compact_hits))
print(
    "Verified direct representation files:",
    sum(len(v) for v in verified_representation.values()),
)
print(
    "Other frozen runtime candidates found in /kaggle/input:",
    len(matches),
)
print()
print("RAW PCAP RECREATION      : PROHIBITED")
print("MODEL RETRAINING         : PROHIBITED")
print("MODEL CONVERSION         : PROHIBITED")
print("BACKEND FORCING          : PROHIBITED")
print("GPU PROFILING STARTED    : FALSE")
print()
print(
    "NEXT: paste the COMPLETE output. "
    "The next cell will restore/verify only the exact required frozen assets "
    "and determine native GPU backend availability per target."
)


STAGE26-G0A :: FRESH RUNTIME IDENTITY
timestamp_utc : 2026-08-20T10:28:23.604634+00:00
hostname      : c386205d4b1c
python        : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
executable    : /usr/bin/python3
platform      : Linux-6.12.90+-x86_64-with-glibc2.35
machine       : x86_64
working dir   : /kaggle/working
/kaggle/input : True
/kaggle/working: True

STAGE26-G0A :: GPU / CUDA HARD GATE
nvidia-smi return code: 0
GPU count: 2
GPU[0] : Tesla T4, GPU-90304bc9-f08d-65c8-0fa2-10d0586c6539, 580.159.04, 15360
GPU[1] : Tesla T4, GPU-899c600e-794e-3a2e-3c0e-fccc6615ffc4, 580.159.04, 15360
nvidia-smi CUDA line: | NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |

STAGE26-G0A :: FRAMEWORK / BACKEND IDENTITY
numpy     : 2.0.2
pandas    : 2.3.3
sklearn   : 1.6.1
xgboost   : 3.2.0
lightgbm  : 4.6.0
catboost  : 1.2.10
torch     : 2.10.0+cu128

torch.cuda.is_available : True
torch.version.cuda      : 12.8
torch.backends.cudnn    : 91002
torch CUDA

In [2]:
# ==================================================================================================
# STAGE26-G0B :: FAST MODEL-ASSET + NATIVE GPU BACKEND GATE
#
# GPU budget is limited. This cell deliberately avoids corpus restoration.
# It verifies the frozen model package, proves deterministic inference inputs can be built
# directly from the frozen repository, probes XGBoost / CatBoost / FT CUDA execution,
# and discovers the exact frozen CNN/ViT loader definitions needed for G1.
#
# NO PERFORMANCE TIMING.
# NO RETRAINING.
# NO MODEL CONVERSION.
# NO RAW PCAP.
# NO COMMIT.
# ==================================================================================================

from __future__ import annotations

import ast
import csv
import hashlib
import importlib.util
import inspect
import json
import subprocess
import sys
from pathlib import Path

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")

EXPECTED_HEAD = "304e5613627a744cbd5d369857f8ac5667a520eb"

RESOLVED = (
    REPO
    / "results/stage26_deployment_profiling/stage26_0_protocol_lock/"
      "stage26_0b_resolved_artifacts.csv"
)

WORKER = (
    REPO
    / "results/stage26_deployment_profiling/stage26_0d_cpu_preflight/"
      "stage26_cpu_preflight_worker.py"
)

SEED = 26042


def banner(s):
    print()
    print("=" * 120)
    print(s)
    print("=" * 120)


def sha256_file(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)

    return h.hexdigest()


def run(cmd):
    p = subprocess.run(
        cmd,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    return p.returncode, p.stdout.strip()


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, str(path))

    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot import {path}")

    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    return mod


# ==================================================================================================
# 1. HARD STATE GATE
# ==================================================================================================

banner("STAGE26-G0B :: HARD STATE GATE")

rc, head = run(["git", "rev-parse", "HEAD"])
rc2, remote = run(["git", "rev-parse", "origin/main"])
rc3, status = run(["git", "status", "--porcelain"])

print("Expected HEAD :", EXPECTED_HEAD)
print("Local HEAD    :", head)
print("origin/main   :", remote)
print("Repo clean    :", status == "")

if (
    rc != 0
    or rc2 != 0
    or rc3 != 0
    or head != EXPECTED_HEAD
    or remote != EXPECTED_HEAD
    or status
):
    raise RuntimeError("Frozen repository state gate failed.")


import torch

print()
print("torch version        :", torch.__version__)
print("CUDA available       :", torch.cuda.is_available())
print("CUDA device count    :", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable.")


for i in range(torch.cuda.device_count()):
    print(
        f"GPU[{i}]              :",
        torch.cuda.get_device_name(i),
    )


# Use GPU 0 only for frozen single-device profiling.
torch.cuda.set_device(0)
DEVICE = torch.device("cuda:0")

print("Profiling device      :", DEVICE)


# ==================================================================================================
# 2. VERIFY COMPLETE FROZEN MODEL ARTIFACT PACKAGE
# ==================================================================================================

banner("STAGE26-G0B :: FROZEN MODEL ARTIFACT BYTE VERIFICATION")

with RESOLVED.open("r", encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f))


wanted_families = {
    "xgboost",
    "lightgbm",
    "catboost",
    "ft_transformer",
    "cnn",
    "vit",
}

model_rows = [
    row
    for row in rows
    if row["family"] in wanted_families
]


print("Resolved model artifact rows:", len(model_rows))

if len(model_rows) != 10:
    raise RuntimeError(
        f"Expected 10 frozen model artifacts, got {len(model_rows)}"
    )


verified = []


for row in model_rows:

    rel = row["repo_relative_path"]

    path = REPO / rel

    if not path.is_file():
        raise RuntimeError(f"Missing frozen artifact: {rel}")

    size = path.stat().st_size

    expected_size = int(float(row["size_bytes"]))

    actual_sha = sha256_file(path)

    expected_sha = row["sha256"].strip()

    ok = (
        size == expected_size
        and actual_sha == expected_sha
    )

    print()
    print("artifact :", row["artifact_id"])
    print("family   :", row["family"])
    print("path     :", rel)
    print("size     :", size, "/", expected_size)
    print("SHA      :", actual_sha)
    print("verified :", ok)

    if not ok:
        raise RuntimeError(
            f"Frozen artifact verification failed: {row['artifact_id']}"
        )

    verified.append(row["artifact_id"])


print()
print("Frozen model artifacts verified:", len(verified), "/ 10")


# ==================================================================================================
# 3. IMPORT FROZEN STAGE26 INPUT/LOADER IMPLEMENTATION
# ==================================================================================================

banner("STAGE26-G0B :: FROZEN INPUT IMPLEMENTATION")

worker = load_module(
    "stage26_cpu_preflight_worker_frozen",
    WORKER,
)


required_worker_functions = [
    "build_group_a_input",
    "build_packet_input",
    "load_ft_model",
]


for name in required_worker_functions:
    print(name, ":", hasattr(worker, name))

    if not hasattr(worker, name):
        raise RuntimeError(
            f"Frozen worker missing required function: {name}"
        )


# --------------------------------------------------------------------------------------------------
# Build only B=1 deterministic inputs to prove corpus-independent inference path.
# --------------------------------------------------------------------------------------------------

group_a = worker.build_group_a_input(
    repo=REPO,
    batch_size=1,
    seed=SEED,
)

packet = worker.build_packet_input(
    repo=REPO,
    batch_size=1,
    seed=SEED,
)


print()
print("GROUP A deterministic input")
print("  derived seed :", group_a["derived_seed"])
print("  X_raw shape  :", group_a["X_raw"].shape)
print("  X_raw dtype  :", group_a["X_raw"].dtype)
print("  Z shape      :", group_a["Z"].shape)


print()
print("GROUP B deterministic input")
print("  derived seed :", packet["derived_seed"])
print("  image shape  :", packet["image_scaled"].shape)
print("  image dtype  :", packet["image_scaled"].dtype)
print("  mask shape   :", packet["padding_mask"].shape)


print()
print("Compact corpus required for warm inference: NO")
print("Raw PCAP accessed                         : NO")


# ==================================================================================================
# 4. XGBOOST NATIVE CUDA PROBE
# ==================================================================================================

banner("STAGE26-G0B :: XGBOOST CUDA PROBE")

import joblib
import numpy as np
import xgboost as xgb


X_raw = group_a["X_raw"]


xgb_path = (
    REPO
    / "results/stage16_classical_benchmark_checkpoint/"
      "stage16_3_tuned_models/XGBOOST_tuned.joblib"
)


xgb_model = joblib.load(xgb_path)

xgb_result = {
    "status": "BACKEND_UNAVAILABLE",
    "detail": None,
}


try:
    import cupy as cp

    X_cp = cp.asarray(X_raw)

    if hasattr(xgb_model, "set_params"):
        xgb_model.set_params(
            device="cuda:0"
        )

    pred = xgb_model.predict_proba(X_cp)[:, 1]

    # Force completion.
    cp.cuda.Stream.null.synchronize()

    pred_np = cp.asnumpy(
        pred
        if isinstance(pred, cp.ndarray)
        else cp.asarray(pred)
    )

    if pred_np.shape != (1,):
        raise RuntimeError(
            f"Unexpected XGB output shape {pred_np.shape}"
        )

    if not np.isfinite(pred_np).all():
        raise RuntimeError("Non-finite XGB prediction.")

    xgb_result = {
        "status": "GPU_NATIVE_AVAILABLE",
        "detail": f"cupy={cp.__version__}; xgboost={xgb.__version__}",
    }

except Exception as e:
    xgb_result = {
        "status": "BACKEND_UNAVAILABLE",
        "detail": f"{type(e).__name__}: {e}",
    }


print("XGBoost:", xgb_result)


# ==================================================================================================
# 5. LIGHTGBM GPU-INFERENCE CAPABILITY CLASSIFICATION
# ==================================================================================================

banner("STAGE26-G0B :: LIGHTGBM GPU-INFERENCE CAPABILITY")

import lightgbm as lgb


lgb_path = (
    REPO
    / "results/stage16_classical_benchmark_checkpoint/"
      "stage16_3_tuned_models/LIGHTGBM_tuned.joblib"
)


lgb_model = joblib.load(lgb_path)


print("LightGBM version :", lgb.__version__)
print("Model class      :", type(lgb_model))
print(
    "predict_proba signature:",
    inspect.signature(lgb_model.predict_proba),
)


# Stage16 recorded GPU TRAINING configuration, but Stage26 GPU profiling requires a native
# GPU INFERENCE path. We do not treat ordinary CPU predict_proba as GPU inference.
#
# No conversion / Treelite / ONNX / custom backend is allowed by the frozen contract.
lgb_result = {
    "status": "BACKEND_UNAVAILABLE",
    "detail": (
        "Frozen LightGBM model has no contract-approved native GPU inference path; "
        "training device_type='gpu' does not authorize CPU prediction to be labeled GPU."
    ),
}


print("LightGBM:", lgb_result)


# ==================================================================================================
# 6. CATBOOST NATIVE GPU PROBE
# ==================================================================================================

banner("STAGE26-G0B :: CATBOOST GPU PROBE")

import catboost


cat_path = (
    REPO
    / "results/stage16_classical_benchmark_checkpoint/"
      "stage16_3_tuned_models/CATBOOST_tuned.joblib"
)


cat_model = joblib.load(cat_path)


cat_result = {
    "status": "BACKEND_UNAVAILABLE",
    "detail": None,
}


try:

    pred = cat_model.predict_proba(
        X_raw,
        task_type="GPU",
    )[:, 1]

    # CatBoost call is synchronous from Python perspective; CUDA sync additionally
    # prevents an async kernel from escaping the smoke probe.
    torch.cuda.synchronize()

    pred = np.asarray(pred)

    if pred.shape != (1,):
        raise RuntimeError(
            f"Unexpected CatBoost output shape {pred.shape}"
        )

    if not np.isfinite(pred).all():
        raise RuntimeError("Non-finite CatBoost prediction.")

    cat_result = {
        "status": "GPU_NATIVE_AVAILABLE",
        "detail": f"catboost={catboost.__version__}",
    }

except Exception as e:

    cat_result = {
        "status": "BACKEND_UNAVAILABLE",
        "detail": f"{type(e).__name__}: {e}",
    }


print("CatBoost:", cat_result)


# ==================================================================================================
# 7. FT TRANSFORMER CUDA PROBE
# ==================================================================================================

banner("STAGE26-G0B :: FT TRANSFORMER CUDA PROBE")


ft_seed7 = (
    REPO
    / "results/stage15_transformer_checkpoint/"
      "stage15_4b_models/FT_BALANCED_seed_7_best_extended.pt"
)


ft_result = {
    "status": "BACKEND_UNAVAILABLE",
    "detail": None,
}


try:

    ft_model, parameter_count, load_mode, state_layout = worker.load_ft_model(
        repo=REPO,
        checkpoint_path=ft_seed7,
    )

    ft_model = ft_model.to(
        DEVICE
    )

    Z_gpu = torch.as_tensor(
        group_a["Z"],
        dtype=torch.float32,
        device=DEVICE,
    )

    torch.cuda.synchronize()

    with torch.inference_mode():
        logits = ft_model(
            Z_gpu
        )

        probs = torch.sigmoid(
            logits.reshape(-1)
        )

    torch.cuda.synchronize()

    probs_cpu = probs.detach().cpu().numpy()

    if probs_cpu.shape != (1,):
        raise RuntimeError(
            f"Unexpected FT output shape {probs_cpu.shape}"
        )

    if not np.isfinite(probs_cpu).all():
        raise RuntimeError("Non-finite FT prediction.")

    ft_result = {
        "status": "GPU_NATIVE_AVAILABLE",
        "detail": {
            "parameters": parameter_count,
            "load_mode": load_mode,
            "state_layout": state_layout,
            "device": str(next(ft_model.parameters()).device),
        },
    }

    del probs
    del logits
    del Z_gpu
    del ft_model

    torch.cuda.empty_cache()

except Exception as e:

    ft_result = {
        "status": "BACKEND_UNAVAILABLE",
        "detail": f"{type(e).__name__}: {e}",
    }


print("FT Transformer:", ft_result)


# ==================================================================================================
# 8. DISCOVER EXACT CNN / VIT FROZEN LOADER DEFINITIONS
# ==================================================================================================

banner("STAGE26-G0B :: CNN / VIT IMPLEMENTATION DISCOVERY")


worker_functions = sorted(
    name
    for name, obj in vars(worker).items()
    if callable(obj)
)


print("Relevant worker callables:")

for name in worker_functions:
    low = name.lower()

    if any(
        token in low
        for token in [
            "cnn",
            "vit",
            "model",
            "torch",
            "packet",
            "mask",
        ]
    ):
        print(" ", name)


# Search only frozen repository Python files for class/function definitions involving CNN/ViT.
candidate_files = []


for root in [
    REPO / "results/stage20_1e_training",
    REPO / "results/stage21_architecture",
    REPO / "scripts",
]:

    if not root.exists():
        continue

    for path in root.rglob("*.py"):

        try:
            text = path.read_text(
                encoding="utf-8"
            )
        except Exception:
            continue

        low = text.lower()

        if (
            "cnn" in low
            or "vit" in low
            or "masked" in low
        ):
            candidate_files.append(path)


print()
print("Candidate implementation files:", len(candidate_files))


definitions = []


for path in sorted(set(candidate_files)):

    try:
        tree = ast.parse(
            path.read_text(
                encoding="utf-8"
            )
        )
    except Exception:
        continue

    defs = []

    for node in ast.walk(tree):

        if isinstance(
            node,
            (
                ast.ClassDef,
                ast.FunctionDef,
            ),
        ):

            low = node.name.lower()

            if any(
                token in low
                for token in [
                    "cnn",
                    "vit",
                    "mask",
                    "model",
                    "network",
                ]
            ):
                defs.append(
                    (
                        type(node).__name__,
                        node.name,
                        node.lineno,
                    )
                )

    if defs:

        rel = path.relative_to(REPO)

        print()
        print(rel)

        for item in defs:
            print(
                f"  {item[0]:12s} "
                f"{item[1]:45s} "
                f"line={item[2]}"
            )

        definitions.append(
            {
                "path": str(rel),
                "definitions": defs,
            }
        )


# ==================================================================================================
# 9. BACKEND SUMMARY
# ==================================================================================================

banner("STAGE26-G0B COMPLETE :: FAST GPU READINESS SUMMARY")


backend_summary = {
    "STAGE16_XGBOOST_TUNED":
        xgb_result["status"],

    "STAGE16_LIGHTGBM_TUNED":
        lgb_result["status"],

    "STAGE16_CATBOOST_TUNED":
        cat_result["status"],

    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING":
        ft_result["status"],

    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE":
        ft_result["status"],

    "STAGE20_MASKED_CNN_V1":
        "TORCH_CUDA_AVAILABLE_LOADER_TO_FREEZE_FROM_OUTPUT",

    "STAGE21_MASKED_VIT_V1":
        "TORCH_CUDA_AVAILABLE_LOADER_TO_FREEZE_FROM_OUTPUT",

    "ENS_LGBM_XGB_EQUAL":
        (
            "BACKEND_UNAVAILABLE"
            if lgb_result["status"] != "GPU_NATIVE_AVAILABLE"
            else xgb_result["status"]
        ),
}


for target, state in backend_summary.items():
    print(
        f"{target:45s} : {state}"
    )


print()
print("Model artifacts verified       :", len(verified), "/ 10")
print("Compact corpus restored        : NO")
print("Compact corpus needed for warm : NO")
print("Raw PCAP accessed              : NO")
print("Timing performed               : NO")
print("GPU memory benchmark performed : NO")
print("Retraining                     : NO")
print("Model conversion               : NO")
print("Repo modified                  :", bool(run(["git", "status", "--porcelain"])[1]))
print()
print("NEXT: paste COMPLETE output.")
print(
    "Then run one consolidated G1 GPU profiling cell "
    "for every contract-supported native GPU target."
)


STAGE26-G0B :: HARD STATE GATE
Expected HEAD : 304e5613627a744cbd5d369857f8ac5667a520eb
Local HEAD    : 304e5613627a744cbd5d369857f8ac5667a520eb
origin/main   : 304e5613627a744cbd5d369857f8ac5667a520eb
Repo clean    : True

torch version        : 2.10.0+cu128
CUDA available       : True
CUDA device count    : 2
GPU[0]              : Tesla T4
GPU[1]              : Tesla T4
Profiling device      : cuda:0

STAGE26-G0B :: FROZEN MODEL ARTIFACT BYTE VERIFICATION
Resolved model artifact rows: 10

artifact : STAGE16_XGBOOST_TUNED
family   : xgboost
path     : results/stage16_classical_benchmark_checkpoint/stage16_3_tuned_models/XGBOOST_tuned.joblib
size     : 864264 / 864264
SHA      : 4f02e06fc6d855adc1e33ce47befda2bedb6c2a5b528338cc88f24e4184b705f
verified : True

artifact : STAGE16_LIGHTGBM_TUNED
family   : lightgbm
path     : results/stage16_classical_benchmark_checkpoint/stage16_3_tuned_models/LIGHTGBM_tuned.joblib
size     : 2420416 / 2420416
SHA      : 0e13bc2386e30a31aba765501011bd5b

In [3]:
from __future__ import annotations

import csv
import gc
import hashlib
import importlib.util
import json
import math
import os
import random
import shutil
import stat
import subprocess
import sys
import tempfile
import textwrap
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

# ==================================================================================================
# STAGE26-G1 :: CONSOLIDATED SINGLE-T4 GPU WARM-INFERENCE + GPU-MEMORY PROFILE
# ==================================================================================================

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")
EXPECTED_PARENT = "304e5613627a744cbd5d369857f8ac5667a520eb"
OUT = REPO / "results/stage26_deployment_profiling/stage26_g1_gpu_warm_profile"
RUNTIME = Path("/kaggle/working/stage26_g1_runtime")
WORKER = RUNTIME / "stage26_g1_worker.py"
COMMIT_MSG = "stage26: profile frozen models on single T4 GPU"

# Physical GPU 1 is selected prospectively before the first G1 measurement because G0B smoke probes
# touched physical GPU 0. This is contamination avoidance, NOT performance-based GPU selection.
PHYSICAL_GPU_INDEX = 1
GPU_LOGICAL_DEVICE = 0  # inside worker because CUDA_VISIBLE_DEVICES=<PHYSICAL_GPU_INDEX>

SEED = 26042
BOOTSTRAP_REPS = 2000
TIMEOUT_SECONDS = 600
MEMORY_SAMPLE_INTERVAL_SECONDS = 0.005
MEMORY_RUN_REPETITIONS = 3
COOLDOWN_SECONDS = 2

TARGETS = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
    "ENS_LGBM_XGB_EQUAL",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
]
SUPPORTED = {
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
}
UNAVAILABLE_DETAIL = {
    "STAGE16_LIGHTGBM_TUNED": (
        "Frozen LightGBM model has no contract-approved native GPU inference path; "
        "GPU training configuration does not make ordinary predict_proba GPU inference."
    ),
    "ENS_LGBM_XGB_EQUAL": (
        "Operational ensemble requires both frozen LightGBM and XGBoost probability generation; "
        "LightGBM native GPU inference is unavailable, so the complete ensemble GPU backend is unavailable."
    ),
}
BATCHES = [1, 64, 256, 1024, 8192]
RUNS = {
    1: {"warmup": 50, "timed": 200},
    64: {"warmup": 30, "timed": 150},
    256: {"warmup": 20, "timed": 100},
    1024: {"warmup": 10, "timed": 50},
    8192: {"warmup": 5, "timed": 20},
}
GROUP = {
    "STAGE16_XGBOOST_TUNED": "GROUP_A_DUPSAFE70",
    "STAGE16_LIGHTGBM_TUNED": "GROUP_A_DUPSAFE70",
    "STAGE16_CATBOOST_TUNED": "GROUP_A_DUPSAFE70",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING": "GROUP_A_DUPSAFE70",
    "STAGE20_MASKED_CNN_V1": "GROUP_B_PACKET_IMAGE",
    "STAGE21_MASKED_VIT_V1": "GROUP_B_PACKET_IMAGE",
    "ENS_LGBM_XGB_EQUAL": "GROUP_A_DUPSAFE70",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE": "GROUP_A_DUPSAFE70",
}

MEASUREMENT_PROTOCOL_SHA256 = "d79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625"
GPU_CONTRACT_SHA256 = "09dfeb386f462597d4fca76aad5676d20901017c6f8175d221d2f85e108d59b7"


def banner(s: str) -> None:
    print("\n" + "=" * 124)
    print(s)
    print("=" * 124)


def run(cmd, *, cwd=REPO, env=None, check=True, timeout=None):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
        timeout=timeout,
    )
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}\n{p.stdout}")
    return p


def git(*args, env=None, check=True):
    return run(["git", *args], cwd=REPO, env=env, check=check).stdout.strip()


def sha256_file(path: Path, block=16 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()


def atomic_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True, allow_nan=False)
        f.write("\n")
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def write_csv(path: Path, fieldnames, rows) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with tmp.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="raise")
        w.writeheader()
        w.writerows(rows)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def q(x, p):
    return float(np.quantile(np.asarray(x, dtype=np.float64), p))


def bootstrap_ci(values, statistic, *, rng, reps=BOOTSTRAP_REPS):
    a = np.asarray(values, dtype=np.float64)
    n = a.size
    if n == 0:
        return None, None
    idx = rng.integers(0, n, size=(reps, n), endpoint=False)
    resampled = a[idx]
    if statistic == "median":
        stats = np.median(resampled, axis=1)
    elif statistic == "p95":
        stats = np.quantile(resampled, 0.95, axis=1)
    elif statistic == "p99":
        stats = np.quantile(resampled, 0.99, axis=1)
    else:
        raise ValueError(statistic)
    return float(np.quantile(stats, 0.025)), float(np.quantile(stats, 0.975))


WORKER_SOURCE = r'''
from __future__ import annotations

import gc
import hashlib
import importlib.util
import json
import os
import subprocess
import sys
import threading
import time
from pathlib import Path

import numpy as np


def atomic_json(path: Path, obj):
    tmp = Path(str(path) + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True, allow_nan=False)
        f.write("\n")
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def import_path(name, path):
    spec = importlib.util.spec_from_file_location(name, str(path))
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Unable to import {path}")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


def extract_torch_state(obj):
    import torch
    if isinstance(obj, dict) and obj and all(torch.is_tensor(v) for v in obj.values()):
        return obj, "DIRECT_STATE_DICT"
    if isinstance(obj, dict):
        for key in ["model_state_dict", "state_dict", "model_state", "network_state_dict", "model"]:
            cand = obj.get(key)
            if isinstance(cand, dict) and cand and all(torch.is_tensor(v) for v in cand.values()):
                return cand, key
    raise RuntimeError("Unable to identify torch state_dict layout")


def torch_load_state(path):
    import torch
    try:
        obj = torch.load(str(path), map_location="cpu", weights_only=True)
        mode = "torch.load(weights_only=True)"
    except TypeError:
        obj = torch.load(str(path), map_location="cpu")
        mode = "torch.load(default)"
    state, layout = extract_torch_state(obj)
    return state, mode, layout


def sha256_array(a):
    x = np.ascontiguousarray(a)
    h = hashlib.sha256()
    h.update(str(x.dtype).encode("ascii"))
    h.update(json.dumps(list(x.shape)).encode("ascii"))
    h.update(x.tobytes(order="C"))
    return h.hexdigest()


def validate_output(p, batch):
    x = np.asarray(p).reshape(-1)
    if x.shape != (int(batch),):
        raise RuntimeError(f"Unexpected output shape {x.shape}, expected {(int(batch),)}")
    if not np.isfinite(x).all():
        raise RuntimeError("Non-finite prediction")
    if (x < 0).any() or (x > 1).any():
        raise RuntimeError("Probability outside [0,1]")
    return x


def gpu_mem_bytes(physical_index):
    pid = os.getpid()
    try:
        import pynvml
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(int(physical_index))
        procs = pynvml.nvmlDeviceGetComputeRunningProcesses(handle)
        total = 0
        for proc in procs:
            if int(proc.pid) == pid:
                used = getattr(proc, "usedGpuMemory", 0)
                if used is not None and int(used) >= 0:
                    total += int(used)
        return total
    except Exception:
        p = subprocess.run(
            [
                "nvidia-smi", "-i", str(physical_index),
                "--query-compute-apps=pid,used_gpu_memory",
                "--format=csv,noheader,nounits",
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            text=True,
            check=False,
        )
        total = 0
        if p.returncode == 0:
            for line in p.stdout.splitlines():
                parts = [z.strip() for z in line.split(",")]
                if len(parts) >= 2:
                    try:
                        if int(parts[0]) == pid:
                            total += int(float(parts[1])) * 1024 * 1024
                    except Exception:
                        pass
        return total


class MemSampler:
    def __init__(self, physical_index, interval):
        self.physical_index = int(physical_index)
        self.interval = float(interval)
        self.values = []
        self.stop_event = threading.Event()
        self.thread = None

    def _run(self):
        while not self.stop_event.is_set():
            try:
                self.values.append(gpu_mem_bytes(self.physical_index))
            except Exception:
                pass
            self.stop_event.wait(self.interval)

    def start(self):
        self.values = [gpu_mem_bytes(self.physical_index)]
        self.stop_event.clear()
        self.thread = threading.Thread(target=self._run, daemon=True)
        self.thread.start()

    def stop(self):
        self.stop_event.set()
        if self.thread is not None:
            self.thread.join(timeout=2.0)
        self.values.append(gpu_mem_bytes(self.physical_index))
        return max(self.values) if self.values else 0


def classify_exception(exc):
    text = f"{type(exc).__name__}: {exc}".lower()
    oom_tokens = ["out of memory", "cuda error: out of memory", "cuda out of memory", "bad_alloc", "memory allocation"]
    if any(t in text for t in oom_tokens):
        return "RESOURCE_LIMIT_OOM"
    try:
        import torch
        if isinstance(exc, torch.cuda.OutOfMemoryError):
            return "RESOURCE_LIMIT_OOM"
    except Exception:
        pass
    return "ERROR"


def main():
    config = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
    out_path = Path(sys.argv[2])

    repo = Path(config["repo"])
    target = config["target_id"]
    batch = int(config["batch_size"])
    seed = int(config["seed"])
    warmup = int(config["warmup_runs"])
    timed = int(config["timed_runs"])
    physical_gpu_index = int(config["physical_gpu_index"])
    mem_interval = float(config["memory_sample_interval_seconds"])
    memory_reps = int(config["memory_run_repetitions"])

    result = {
        "condition_id": config["condition_id"],
        "execution_index": int(config["execution_index"]),
        "target_id": target,
        "comparison_group": config["comparison_group"],
        "batch_size": batch,
        "status": "STARTED",
        "warmup_runs_planned": warmup,
        "timed_runs_planned": timed,
        "timed_runs_completed": 0,
        "elapsed_ns": [],
        "physical_gpu_index": physical_gpu_index,
        "logical_cuda_device": 0,
        "timing_boundary": "PREPARED_MODEL_INPUT_TO_HOST_NUMPY_ATTACK_PROBABILITY",
        "gpu_sync_before_after_every_timed_region": True,
        "memory_run_separate_from_timing": True,
        "memory_sample_interval_seconds": mem_interval,
        "memory_run_repetitions": memory_reps,
        "raw_pcap_accessed": False,
        "compact_corpus_accessed": False,
        "retraining": False,
        "model_conversion": False,
        "error": None,
    }

    try:
        # Frozen preflight implementation provides deterministic model inputs and FT loader.
        worker = import_path(
            "stage26_frozen_preflight_impl",
            repo / "results/stage26_deployment_profiling/stage26_0d_cpu_preflight/stage26_cpu_preflight_worker.py",
        )

        def sync_cupy():
            import cupy as cp
            cp.cuda.runtime.deviceSynchronize()

        def sync_torch():
            import torch
            torch.cuda.synchronize(0)

        model = None
        models = None
        input_objects = []
        framework = None
        sync_device = None
        infer_once = None
        torch_model = False
        load_receipt = None
        parameter_count = None
        output_boundary_detail = None

        # ------------------------------- XGBoost --------------------------------
        if target == "STAGE16_XGBOOST_TUNED":
            import cupy as cp
            import joblib
            import xgboost
            cp.cuda.Device(0).use()
            # Establish CUDA context before baseline memory observation.
            _tmp = cp.empty((1,), dtype=cp.uint8)
            sync_cupy()
            del _tmp
            cp.get_default_memory_pool().free_all_blocks()
            baseline = gpu_mem_bytes(physical_gpu_index)

            model = joblib.load(repo / "results/stage16_classical_benchmark_checkpoint/stage16_3_tuned_models/XGBOOST_tuned.joblib")
            if hasattr(model, "set_params"):
                model.set_params(device="cuda:0", n_jobs=1)
            loaded = gpu_mem_bytes(physical_gpu_index)

            inp = worker.build_group_a_input(repo=repo, batch_size=batch, seed=seed)
            X_dev = cp.asarray(inp["X_raw"])
            sync_cupy()
            input_objects = [X_dev]
            prepared = gpu_mem_bytes(physical_gpu_index)
            framework = f"xgboost={xgboost.__version__};cupy={cp.__version__}"
            sync_device = sync_cupy
            output_boundary_detail = "CUPY_DEVICE_INPUT_TO_HOST_NUMPY_PROBABILITY"

            def infer_once():
                pred = model.predict_proba(X_dev)[:, 1]
                if isinstance(pred, cp.ndarray):
                    return cp.asnumpy(pred)
                return np.asarray(pred)

        # ------------------------------- CatBoost --------------------------------
        elif target == "STAGE16_CATBOOST_TUNED":
            import cupy as cp
            import joblib
            import catboost
            cp.cuda.Device(0).use()
            _tmp = cp.empty((1,), dtype=cp.uint8)
            sync_cupy()
            del _tmp
            cp.get_default_memory_pool().free_all_blocks()
            baseline = gpu_mem_bytes(physical_gpu_index)

            model = joblib.load(repo / "results/stage16_classical_benchmark_checkpoint/stage16_3_tuned_models/CATBOOST_tuned.joblib")
            loaded = gpu_mem_bytes(physical_gpu_index)

            inp = worker.build_group_a_input(repo=repo, batch_size=batch, seed=seed)
            X_host = np.ascontiguousarray(inp["X_raw"], dtype=np.float32)
            input_objects = [X_host]
            prepared = gpu_mem_bytes(physical_gpu_index)
            framework = f"catboost={catboost.__version__}"
            sync_device = sync_cupy
            output_boundary_detail = "HOST_NUMPY_INPUT_NATIVE_CATBOOST_GPU_TO_HOST_NUMPY_PROBABILITY"

            def infer_once():
                return np.asarray(model.predict_proba(X_host, task_type="GPU")[:, 1])

        # ------------------------------- FT --------------------------------
        elif target in {"FT_BALANCED_5_CHECKPOINT_SOFT_VOTING", "FT_BALANCED_SINGLE_RESOURCE_REFERENCE"}:
            import torch
            torch.cuda.set_device(0)
            _tmp = torch.empty((1,), device="cuda:0")
            sync_torch()
            del _tmp
            torch.cuda.empty_cache()
            baseline = gpu_mem_bytes(physical_gpu_index)

            seed_paths = {
                7: repo / "results/stage15_transformer_checkpoint/stage15_4b_models/FT_BALANCED_seed_7_best_extended.pt",
                29: repo / "results/stage15_transformer_checkpoint/stage15_4a_models/FT_BALANCED_seed_29_best.pt",
                101: repo / "results/stage15_transformer_checkpoint/stage15_4a_models/FT_BALANCED_seed_101_best.pt",
                313: repo / "results/stage15_transformer_checkpoint/stage15_4c_models/FT_BALANCED_seed_313_best.pt",
                997: repo / "results/stage15_transformer_checkpoint/stage15_4c_models/FT_BALANCED_seed_997_best.pt",
            }
            seeds = [7, 29, 101, 313, 997] if target == "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING" else [7]
            models = []
            receipts = []
            for checkpoint_seed in seeds:
                m, pc, mode, layout = worker.load_ft_model(repo=repo, checkpoint_path=seed_paths[checkpoint_seed])
                m = m.to("cuda:0")
                m.eval()
                models.append(m)
                receipts.append({"seed": checkpoint_seed, "parameter_count": pc, "load_mode": mode, "state_layout": layout})
            sync_torch()
            loaded = gpu_mem_bytes(physical_gpu_index)

            inp = worker.build_group_a_input(repo=repo, batch_size=batch, seed=seed)
            x = torch.as_tensor(inp["Z"], dtype=torch.float32, device="cuda:0")
            sync_torch()
            input_objects = [x]
            prepared = gpu_mem_bytes(physical_gpu_index)
            framework = f"torch={torch.__version__}"
            sync_device = sync_torch
            torch_model = True
            load_receipt = receipts
            parameter_count = 159169
            output_boundary_detail = "CUDA_TENSOR_TO_PER_MEMBER_HOST_NUMPY_PROBABILITIES_THEN_HOST_MEAN"

            def infer_once():
                member = []
                with torch.inference_mode():
                    for m in models:
                        logits = m(x)
                        p = torch.sigmoid(logits).detach().cpu().numpy()
                        member.append(p)
                return np.mean(np.stack(member, axis=0), axis=0)

        # ------------------------------- CNN / ViT --------------------------------
        elif target in {"STAGE20_MASKED_CNN_V1", "STAGE21_MASKED_VIT_V1"}:
            import torch
            torch.cuda.set_device(0)
            _tmp = torch.empty((1,), device="cuda:0")
            sync_torch()
            del _tmp
            torch.cuda.empty_cache()
            baseline = gpu_mem_bytes(physical_gpu_index)

            if target == "STAGE20_MASKED_CNN_V1":
                module = import_path("stage26_cnn", repo / "scripts/stage20_masked_cnn.py")
                model = module.Stage20MaskedCNNv1()
                state, mode, layout = torch_load_state(repo / "results/stage20_1e_training/stage20_1e2_epoch10_model_state_dict.pt")
                model.load_state_dict(state, strict=True)
                parameter_count = int(module.count_trainable_parameters(model))
                if parameter_count != 93025:
                    raise RuntimeError(f"CNN parameter count mismatch {parameter_count}")
                output_boundary_detail = "CUDA_UINT8_IMAGE_CNN_INTERNAL_FLOAT32_DIV255_TO_HOST_NUMPY_PROBABILITY"
            else:
                module = import_path("stage26_vit", repo / "scripts/stage21_masked_vit.py")
                model = module.Stage21MaskedViTv1()
                state, mode, layout = torch_load_state(repo / "results/stage21_architecture/stage21_2_epoch10_model_state_dict.pt")
                model.load_state_dict(state, strict=True)
                parameter_count = int(module.count_trainable_parameters(model))
                if parameter_count != 91969:
                    raise RuntimeError(f"ViT parameter count mismatch {parameter_count}")
                output_boundary_detail = "CUDA_FLOAT32_DIV255_IMAGE_TO_HOST_NUMPY_PROBABILITY"

            model = model.to("cuda:0")
            model.eval()
            sync_torch()
            loaded = gpu_mem_bytes(physical_gpu_index)

            # Frozen packet-image preparation is outside T_infer. GPU input tensors are created only
            # after the model-loaded memory observation, so loaded vs prepared remain interpretable.
            inp = worker.build_packet_input(repo=repo, batch_size=batch, seed=seed)
            if target == "STAGE20_MASKED_CNN_V1":
                image = torch.from_numpy(inp["image_uint8"]).to("cuda:0")
            else:
                image = torch.from_numpy(inp["image_scaled"]).to("cuda:0")
            mask = torch.from_numpy(inp["padding_mask"]).to("cuda:0")
            sync_torch()
            prepared = gpu_mem_bytes(physical_gpu_index)
            input_objects = [image, mask]
            framework = f"torch={torch.__version__}"
            sync_device = sync_torch
            torch_model = True
            load_receipt = {"load_mode": mode, "state_layout": layout}

            def infer_once():
                with torch.inference_mode():
                    logits = model(image, mask)
                    return torch.sigmoid(logits).detach().cpu().numpy()

        else:
            raise RuntimeError(f"Unsupported worker target {target}")

        result.update({
            "framework": framework,
            "gpu_memory_baseline_bytes": int(baseline),
            "gpu_memory_loaded_bytes": int(loaded),
            "gpu_memory_prepared_input_bytes": int(prepared),
            "parameter_count": parameter_count,
            "load_receipt": load_receipt,
            "output_boundary_detail": output_boundary_detail,
        })

        # Untimed correctness smoke inference.
        sync_device()
        smoke = validate_output(infer_once(), batch)
        sync_device()
        result["smoke_output_sha256"] = sha256_array(smoke)

        # Warmup: untimed, synchronized.
        for _ in range(warmup):
            sync_device()
            _ = infer_once()
            sync_device()

        # Timed region. Validation/checksum is outside the timer.
        elapsed = []
        last = None
        for _ in range(timed):
            sync_device()
            t0 = time.perf_counter_ns()
            last = infer_once()
            sync_device()
            t1 = time.perf_counter_ns()
            elapsed.append(int(t1 - t0))

        last = validate_output(last, batch)
        result["timed_output_sha256"] = sha256_array(last)
        result["elapsed_ns"] = elapsed
        result["timed_runs_completed"] = len(elapsed)

        # Separate memory run; no timing observations are taken while sampler is active.
        torch_peak_allocated = None
        if torch_model:
            import torch
            torch.cuda.reset_peak_memory_stats(0)

        sampler = MemSampler(physical_gpu_index, mem_interval)
        sampler.start()
        try:
            for _ in range(memory_reps):
                sync_device()
                mem_out = infer_once()
                sync_device()
            validate_output(mem_out, batch)
        finally:
            peak = sampler.stop()

        if torch_model:
            import torch
            torch_peak_allocated = int(torch.cuda.max_memory_allocated(0))

        result["gpu_memory_peak_process_bytes"] = int(peak)
        result["torch_peak_allocated_bytes"] = torch_peak_allocated
        result["status"] = "PASS"

    except Exception as exc:
        result["status"] = classify_exception(exc)
        result["error"] = f"{type(exc).__name__}: {exc}"
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass

    atomic_json(out_path, result)


if __name__ == "__main__":
    main()
'''


# ==================================================================================================
# 0. DURABLE + GPU STATE GATES
# ==================================================================================================

banner("STAGE26-G1 :: DURABLE SCIENTIFIC PARENT GATE")

git("fetch", "origin", "main")
head = git("rev-parse", "HEAD")
remote = git("rev-parse", "origin/main")
status = git("status", "--porcelain")
print("Expected parent:", EXPECTED_PARENT)
print("Local HEAD     :", head)
print("origin/main    :", remote)
print("Repo clean     :", status == "")
if head != EXPECTED_PARENT or remote != EXPECTED_PARENT or status:
    raise RuntimeError("G1 parent/repository gate failed")
if OUT.exists():
    raise RuntimeError(f"G1 durable output already exists: {OUT}")

banner("STAGE26-G1 :: PROSPECTIVE SINGLE-T4 DEVICE LOCK")
q = run([
    "nvidia-smi", "-i", str(PHYSICAL_GPU_INDEX),
    "--query-gpu=index,name,uuid,driver_version,memory.total,memory.free",
    "--format=csv,noheader,nounits",
], check=True)
print(q.stdout.strip())
# Require physical GPU 1 to have no compute process before first G1 measurement.
apps = run([
    "nvidia-smi", "-i", str(PHYSICAL_GPU_INDEX),
    "--query-compute-apps=pid,used_gpu_memory",
    "--format=csv,noheader,nounits",
], check=False)
app_lines = [x for x in apps.stdout.splitlines() if x.strip()]
print("Existing compute processes on selected GPU:", len(app_lines))
for line in app_lines:
    print(" ", line)
if app_lines:
    raise RuntimeError(
        "Selected physical GPU is not pristine before G1. Do not choose another GPU based on performance; "
        "clear the contamination and rerun this exact cell."
    )
print("Selection rationale: physical GPU 1 was not used by G0B smoke probes; selection is contamination avoidance, not performance selection.")

# ==================================================================================================
# 1. WRITE EPHEMERAL WORKER + FREEZE CONDITION PLAN
# ==================================================================================================

banner("STAGE26-G1 :: FREEZE RANDOMIZED GPU CONDITION PLAN")
RUNTIME.mkdir(parents=True, exist_ok=True)
WORKER.write_text(WORKER_SOURCE, encoding="utf-8")
pyc = run([sys.executable, "-m", "py_compile", str(WORKER)], cwd=Path("/kaggle/working"), check=True)
print("Worker py_compile: PASS")
print("Worker SHA256     :", sha256_file(WORKER))

conditions = []
condition_no = 0
for target in TARGETS:
    for batch in BATCHES:
        condition_no += 1
        conditions.append({
            "condition_id": f"GPUCOND_{condition_no:03d}",
            "target_id": target,
            "comparison_group": GROUP[target],
            "batch_size": batch,
            "planned_backend_status": "NATIVE_GPU" if target in SUPPORTED else "BACKEND_UNAVAILABLE",
            "warmup_runs": RUNS[batch]["warmup"],
            "timed_runs": RUNS[batch]["timed"],
        })

rng_plan = np.random.default_rng(SEED)
order = rng_plan.permutation(len(conditions))
execution_plan = []
for execution_index, idx in enumerate(order, start=1):
    row = dict(conditions[int(idx)])
    row["execution_index"] = execution_index
    execution_plan.append(row)

print("Conditions total       :", len(execution_plan))
print("Supported measurements :", sum(x["planned_backend_status"] == "NATIVE_GPU" for x in execution_plan))
print("Backend unavailable    :", sum(x["planned_backend_status"] == "BACKEND_UNAVAILABLE" for x in execution_plan))
print("Randomization seed     :", SEED)
print("Selected physical GPU  :", PHYSICAL_GPU_INDEX)

# ==================================================================================================
# 2. EXECUTE CONDITIONS
# ==================================================================================================

banner("STAGE26-G1 :: EXECUTE RANDOMIZED GPU CONDITIONS")

checkpoint_path = RUNTIME / "stage26_g1_checkpoint.json"
condition_results = []
raw_observations = []

env_base = os.environ.copy()
env_base["CUDA_VISIBLE_DEVICES"] = str(PHYSICAL_GPU_INDEX)
env_base["PYTHONUNBUFFERED"] = "1"
env_base["OMP_NUM_THREADS"] = "1"
env_base["MKL_NUM_THREADS"] = "1"
env_base["OPENBLAS_NUM_THREADS"] = "1"
env_base["NUMEXPR_NUM_THREADS"] = "1"

for plan in execution_plan:
    target = plan["target_id"]
    batch = int(plan["batch_size"])
    cid = plan["condition_id"]
    ex = int(plan["execution_index"])

    print(f"[{ex:02d}/40] {cid}  {target:45s}  B={batch}", flush=True)

    if target not in SUPPORTED:
        res = {
            **plan,
            "status": "BACKEND_UNAVAILABLE",
            "error": UNAVAILABLE_DETAIL[target],
            "timed_runs_completed": 0,
            "elapsed_ns": [],
            "physical_gpu_index": PHYSICAL_GPU_INDEX,
            "timing_boundary": "NOT_APPLICABLE_BACKEND_UNAVAILABLE",
            "gpu_memory_baseline_bytes": None,
            "gpu_memory_loaded_bytes": None,
            "gpu_memory_prepared_input_bytes": None,
            "gpu_memory_peak_process_bytes": None,
            "torch_peak_allocated_bytes": None,
        }
        condition_results.append(res)
        print("    -> BACKEND_UNAVAILABLE")
    else:
        cfg = {
            **plan,
            "repo": str(REPO),
            "seed": SEED,
            "physical_gpu_index": PHYSICAL_GPU_INDEX,
            "memory_sample_interval_seconds": MEMORY_SAMPLE_INTERVAL_SECONDS,
            "memory_run_repetitions": MEMORY_RUN_REPETITIONS,
        }
        cfg_path = RUNTIME / f"{cid}.config.json"
        out_path = RUNTIME / f"{cid}.result.json"
        atomic_json(cfg_path, cfg)
        try:
            p = run(
                [sys.executable, str(WORKER), str(cfg_path), str(out_path)],
                cwd=Path("/kaggle/working"),
                env=env_base,
                check=False,
                timeout=TIMEOUT_SECONDS,
            )
            if not out_path.exists():
                raise RuntimeError(f"Worker produced no result file. returncode={p.returncode}\n{p.stdout}")
            res = json.loads(out_path.read_text(encoding="utf-8"))
            if p.returncode != 0 and res.get("status") == "PASS":
                raise RuntimeError(f"Worker nonzero return with PASS result: {p.returncode}\n{p.stdout}")
        except subprocess.TimeoutExpired:
            res = {
                **plan,
                "status": "TIMEOUT_RESOURCE_LIMIT",
                "error": f"Condition exceeded frozen timeout of {TIMEOUT_SECONDS} seconds",
                "timed_runs_completed": 0,
                "elapsed_ns": [],
                "physical_gpu_index": PHYSICAL_GPU_INDEX,
                "timing_boundary": "PREPARED_MODEL_INPUT_TO_HOST_NUMPY_ATTACK_PROBABILITY",
                "gpu_memory_baseline_bytes": None,
                "gpu_memory_loaded_bytes": None,
                "gpu_memory_prepared_input_bytes": None,
                "gpu_memory_peak_process_bytes": None,
                "torch_peak_allocated_bytes": None,
            }
        condition_results.append(res)
        print(f"    -> {res['status']}  timed={res.get('timed_runs_completed', 0)}  error={res.get('error')}", flush=True)

        if res["status"] == "ERROR":
            atomic_json(checkpoint_path, {"execution_plan": execution_plan, "condition_results": condition_results})
            raise RuntimeError(f"Unexpected G1 implementation/backend error in {cid}: {res.get('error')}")

        if res["status"] == "PASS":
            elapsed = [int(x) for x in res["elapsed_ns"]]
            if len(elapsed) != int(plan["timed_runs"]):
                raise RuntimeError(f"Timed observation count mismatch for {cid}")
            for i, ns in enumerate(elapsed, start=1):
                raw_observations.append({
                    "condition_id": cid,
                    "execution_index": ex,
                    "target_id": target,
                    "comparison_group": plan["comparison_group"],
                    "hardware_mode": "GPU_SINGLE_T4",
                    "physical_gpu_index": PHYSICAL_GPU_INDEX,
                    "batch_size": batch,
                    "iteration_index": i,
                    "elapsed_ns": ns,
                    "batch_latency_ms": ns / 1e6,
                    "flows_per_second": batch * 1e9 / ns,
                })

    atomic_json(checkpoint_path, {"execution_plan": execution_plan, "condition_results": condition_results})
    if ex < len(execution_plan):
        time.sleep(COOLDOWN_SECONDS)

# ==================================================================================================
# 3. SUMMARIZE + PROSPECTIVE BOOTSTRAP
# ==================================================================================================

banner("STAGE26-G1 :: SUMMARIZE TIMING + CORRECT PROSPECTIVE UNCERTAINTY")

by_cid = {}
for row in raw_observations:
    by_cid.setdefault(row["condition_id"], []).append(row)

summary_rows = []
bootstrap_rng = np.random.default_rng(SEED)
for plan in sorted(conditions, key=lambda x: x["condition_id"]):
    cid = plan["condition_id"]
    res = next(x for x in condition_results if x["condition_id"] == cid)
    status_value = res["status"]
    base = {
        "condition_id": cid,
        "target_id": plan["target_id"],
        "comparison_group": plan["comparison_group"],
        "hardware_mode": "GPU_SINGLE_T4",
        "physical_gpu_index": PHYSICAL_GPU_INDEX,
        "batch_size": int(plan["batch_size"]),
        "status": status_value,
        "n": 0,
        "p50_latency_ms": "",
        "p50_ci95_low_ms": "",
        "p50_ci95_high_ms": "",
        "p95_latency_ms": "",
        "p95_ci95_low_ms": "",
        "p95_ci95_high_ms": "",
        "p99_latency_ms_if_n_gte_100": "",
        "p99_ci95_low_ms_if_n_gte_100": "",
        "p99_ci95_high_ms_if_n_gte_100": "",
        "median_throughput_flows_per_second": "",
        "throughput_ci95_low_flows_per_second": "",
        "throughput_ci95_high_flows_per_second": "",
        "gpu_memory_baseline_mib": "",
        "gpu_memory_loaded_mib": "",
        "gpu_memory_prepared_input_mib": "",
        "gpu_memory_peak_process_mib": "",
        "delta_model_gpu_memory_mib": "",
        "delta_peak_gpu_memory_mib": "",
        "torch_peak_allocated_mib": "",
        "timing_boundary": res.get("timing_boundary", ""),
        "output_boundary_detail": res.get("output_boundary_detail", ""),
        "error_or_backend_detail": res.get("error") or "",
    }
    if status_value == "PASS":
        obs = by_cid[cid]
        lat = np.asarray([x["batch_latency_ms"] for x in obs], dtype=np.float64)
        thr = np.asarray([x["flows_per_second"] for x in obs], dtype=np.float64)
        n = len(lat)
        base["n"] = n
        base["p50_latency_ms"] = float(np.median(lat))
        base["p95_latency_ms"] = float(np.quantile(lat, 0.95))
        base["median_throughput_flows_per_second"] = float(np.median(thr))

        # One deterministic bootstrap resample matrix per metric stream using the single frozen RNG stream.
        lo, hi = bootstrap_ci(lat, "median", rng=bootstrap_rng)
        base["p50_ci95_low_ms"] = lo
        base["p50_ci95_high_ms"] = hi
        lo, hi = bootstrap_ci(lat, "p95", rng=bootstrap_rng)
        base["p95_ci95_low_ms"] = lo
        base["p95_ci95_high_ms"] = hi
        if n >= 100:
            base["p99_latency_ms_if_n_gte_100"] = float(np.quantile(lat, 0.99))
            lo, hi = bootstrap_ci(lat, "p99", rng=bootstrap_rng)
            base["p99_ci95_low_ms_if_n_gte_100"] = lo
            base["p99_ci95_high_ms_if_n_gte_100"] = hi
        lo, hi = bootstrap_ci(thr, "median", rng=bootstrap_rng)
        base["throughput_ci95_low_flows_per_second"] = lo
        base["throughput_ci95_high_flows_per_second"] = hi

        def mib(v):
            return "" if v is None else float(v) / (1024 ** 2)

        baseline_b = res.get("gpu_memory_baseline_bytes")
        loaded_b = res.get("gpu_memory_loaded_bytes")
        prepared_b = res.get("gpu_memory_prepared_input_bytes")
        peak_b = res.get("gpu_memory_peak_process_bytes")
        base["gpu_memory_baseline_mib"] = mib(baseline_b)
        base["gpu_memory_loaded_mib"] = mib(loaded_b)
        base["gpu_memory_prepared_input_mib"] = mib(prepared_b)
        base["gpu_memory_peak_process_mib"] = mib(peak_b)
        if baseline_b is not None and loaded_b is not None:
            base["delta_model_gpu_memory_mib"] = (loaded_b - baseline_b) / (1024 ** 2)
        if baseline_b is not None and peak_b is not None:
            base["delta_peak_gpu_memory_mib"] = (peak_b - baseline_b) / (1024 ** 2)
        base["torch_peak_allocated_mib"] = mib(res.get("torch_peak_allocated_bytes"))

    summary_rows.append(base)

status_counts = {}
for r in summary_rows:
    status_counts[r["status"]] = status_counts.get(r["status"], 0) + 1
print("Status counts:", status_counts)
print("Raw timing observations:", len(raw_observations))

# ==================================================================================================
# 4. DURABLE OUTPUT PACKAGE
# ==================================================================================================

banner("STAGE26-G1 :: WRITE DURABLE GPU PACKAGE")
OUT.mkdir(parents=True, exist_ok=False)

plan_csv = OUT / "stage26_g1_gpu_execution_plan.csv"
raw_csv = OUT / "stage26_g1_gpu_raw_timing.csv"
summary_csv = OUT / "stage26_g1_gpu_condition_summary.csv"
worker_results_json = OUT / "stage26_g1_gpu_worker_results.json"
env_json = OUT / "stage26_g1_gpu_environment.json"
receipt_json = OUT / "stage26_g1_gpu_profile_receipt.json"
manifest_json = OUT / "stage26_g1_gpu_profile_manifest.json"

write_csv(
    plan_csv,
    ["execution_index", "condition_id", "target_id", "comparison_group", "batch_size", "planned_backend_status", "warmup_runs", "timed_runs"],
    execution_plan,
)
write_csv(
    raw_csv,
    ["condition_id", "execution_index", "target_id", "comparison_group", "hardware_mode", "physical_gpu_index", "batch_size", "iteration_index", "elapsed_ns", "batch_latency_ms", "flows_per_second"],
    raw_observations,
)
summary_fields = list(summary_rows[0].keys())
write_csv(summary_csv, summary_fields, summary_rows)
atomic_json(worker_results_json, condition_results)

gpu_info = run([
    "nvidia-smi", "-i", str(PHYSICAL_GPU_INDEX),
    "--query-gpu=index,name,uuid,driver_version,memory.total",
    "--format=csv,noheader,nounits",
], check=True).stdout.strip()
try:
    import torch
    torch_version = torch.__version__
    torch_cuda = torch.version.cuda
except Exception:
    torch_version = None
    torch_cuda = None

environment = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "scientific_parent": EXPECTED_PARENT,
    "hardware_mode": "GPU_SINGLE_T4",
    "selected_physical_gpu_index": PHYSICAL_GPU_INDEX,
    "selected_gpu_reason": "GPU1_PRISTINE_BEFORE_G1_GPU0_USED_ONLY_FOR_G0B_SMOKE_PROBES_NOT_PERFORMANCE_SELECTION",
    "nvidia_smi_gpu_record": gpu_info,
    "python": sys.version,
    "torch_version_parent": torch_version,
    "torch_cuda_parent": torch_cuda,
    "measurement_protocol_sha256": MEASUREMENT_PROTOCOL_SHA256,
    "gpu_bootstrap_contract_sha256": GPU_CONTRACT_SHA256,
    "worker_sha256": sha256_file(WORKER),
}
atomic_json(env_json, environment)

receipt = {
    "schema": "stage26_g1_gpu_warm_profile_receipt_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "scientific_parent": EXPECTED_PARENT,
    "hardware_mode": "GPU_SINGLE_T4",
    "selected_physical_gpu_index": PHYSICAL_GPU_INDEX,
    "condition_count": len(summary_rows),
    "supported_condition_count": 30,
    "backend_unavailable_condition_count": 10,
    "status_counts": status_counts,
    "raw_timing_observation_count": len(raw_observations),
    "randomization_seed": SEED,
    "bootstrap": {
        "enabled": True,
        "replicates": BOOTSTRAP_REPS,
        "interval": "PERCENTILE_95_PERCENT",
        "rng": "numpy.random.default_rng_PCG64",
        "seed": SEED,
        "stream_order": "CONDITIONS_SORTED_BY_CONDITION_ID__P50_THEN_P95_THEN_P99_IF_ELIGIBLE_THEN_MEDIAN_THROUGHPUT",
    },
    "timing": {
        "clock": "time.perf_counter_ns",
        "gpu_synchronization": "DEVICE_SYNCHRONIZE_BEFORE_AND_AFTER_EVERY_TIMED_REGION",
        "boundary": "PREPARED_MODEL_INPUT_TO_HOST_NUMPY_ATTACK_PROBABILITY",
        "warmup_and_timed_runs": RUNS,
        "condition_timeout_seconds": TIMEOUT_SECONDS,
        "cooldown_seconds_between_conditions": COOLDOWN_SECONDS,
    },
    "memory": {
        "required": True,
        "timing_and_memory_runs_separate": True,
        "process_gpu_memory_sampler": "NVML_WITH_NVIDIA_SMI_FALLBACK",
        "sampling_interval_seconds": MEMORY_SAMPLE_INTERVAL_SECONDS,
        "memory_run_repetitions": MEMORY_RUN_REPETITIONS,
        "torch_allocator_peak_also_recorded_for_torch_targets": True,
    },
    "backend_policy": {
        "STAGE16_LIGHTGBM_TUNED": "BACKEND_UNAVAILABLE",
        "ENS_LGBM_XGB_EQUAL": "BACKEND_UNAVAILABLE_DUE_TO_REQUIRED_LIGHTGBM_COMPONENT",
        "no_forced_backend_substitution": True,
        "no_model_conversion": True,
        "no_retraining": True,
    },
    "scientific_boundaries": {
        "compact_corpus_required_for_warm_inference": False,
        "raw_pcap_accessed": False,
        "complete_e2e_measurement": "UNAVAILABLE",
        "cross_group_pareto": "PROHIBITED",
        "missing_cost_imputation": "PROHIBITED",
    },
}
atomic_json(receipt_json, receipt)

files_for_manifest = [plan_csv, raw_csv, summary_csv, worker_results_json, env_json, receipt_json]
manifest = {
    "schema": "stage26_g1_gpu_profile_manifest_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "scientific_parent": EXPECTED_PARENT,
    "files": [
        {
            "path": str(p.relative_to(REPO)),
            "sha256": sha256_file(p),
            "size_bytes": p.stat().st_size,
        }
        for p in files_for_manifest
    ],
}
atomic_json(manifest_json, manifest)

for p in files_for_manifest + [manifest_json]:
    print(f"{p.name:44s} {p.stat().st_size:10d}  {sha256_file(p)}")

# ==================================================================================================
# 5. SCIENTIFIC SANITY CHECKS
# ==================================================================================================

banner("STAGE26-G1 :: SCIENTIFIC SANITY CHECKS")
if len(summary_rows) != 40:
    raise RuntimeError("Expected exactly 40 GPU condition rows")
if sum(r["status"] == "BACKEND_UNAVAILABLE" for r in summary_rows) != 10:
    raise RuntimeError("Expected exactly 10 BACKEND_UNAVAILABLE rows")
for r in summary_rows:
    if r["status"] != "PASS":
        metric_fields = [
            "p50_latency_ms", "p50_ci95_low_ms", "p50_ci95_high_ms",
            "p95_latency_ms", "p95_ci95_low_ms", "p95_ci95_high_ms",
            "p99_latency_ms_if_n_gte_100", "p99_ci95_low_ms_if_n_gte_100", "p99_ci95_high_ms_if_n_gte_100",
            "median_throughput_flows_per_second", "throughput_ci95_low_flows_per_second", "throughput_ci95_high_flows_per_second",
        ]
        if any(r[f] != "" for f in metric_fields):
            raise RuntimeError(f"Missing-cost imputation detected in {r['condition_id']}")
    if r["status"] == "PASS" and int(r["n"]) != RUNS[int(r["batch_size"])]["timed"]:
        raise RuntimeError(f"PASS observation count mismatch in {r['condition_id']}")
print("40-row frozen geometry                  : PASS")
print("10 LightGBM-dependent backend-unavailable: PASS")
print("Non-PASS timing/cost imputation          : NONE")
print("GPU timing synchronization               : MANDATORY / APPLIED")
print("Timing and GPU-memory runs               : SEPARATE")
print("Raw PCAP                                 : NOT ACCESSED")
print("Compact corpus                           : NOT REQUIRED / NOT ACCESSED")
print("Retraining / conversion                  : NONE")

# ==================================================================================================
# 6. COMMIT + PUSH + REMOTE BYTE VERIFY
# ==================================================================================================

banner("STAGE26-G1 :: COMMIT")
pre = git("status", "--porcelain")
print(pre)
expected_prefix = "?? " + str(OUT.relative_to(REPO))
lines = [x for x in pre.splitlines() if x.strip()]
if not lines or any(not x.startswith(expected_prefix) for x in lines):
    raise RuntimeError("Unexpected repository modifications before G1 commit")

git("add", str(OUT.relative_to(REPO)))
staged = git("diff", "--cached", "--name-only").splitlines()
expected_staged = sorted(str(p.relative_to(REPO)) for p in files_for_manifest + [manifest_json])
print("Staged files:")
for x in staged:
    print(" ", x)
if sorted(staged) != expected_staged:
    raise RuntimeError("Unexpected staged file set")

git("commit", "-m", COMMIT_MSG)
new_head = git("rev-parse", "HEAD")
parent = git("rev-parse", "HEAD^")
subject = git("log", "-1", "--pretty=%s")
print("Parent :", parent)
print("HEAD   :", new_head)
print("Subject:", subject)
if parent != EXPECTED_PARENT or subject != COMMIT_MSG:
    raise RuntimeError("G1 commit identity failed")

banner("STAGE26-G1 :: PUSH")
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
if not token or len(token.strip()) < 20:
    raise RuntimeError("GITHUB_TOKEN unavailable")
fd, askname = tempfile.mkstemp(prefix="stage26_g1_askpass_", suffix=".sh")
os.close(fd)
ask = Path(askname)
try:
    ask.write_text(
        '#!/bin/sh\ncase "$1" in\n  *Username*) printf "%s\\n" "x-access-token" ;;\n  *Password*) printf "%s\\n" "$STAGE26_GITHUB_TOKEN" ;;\n  *) printf "%s\\n" "" ;;\nesac\n',
        encoding="utf-8",
    )
    ask.chmod(ask.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(ask)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["STAGE26_GITHUB_TOKEN"] = token.strip()
    print(git("push", "origin", "main", env=env))
finally:
    ask.unlink(missing_ok=True)
    token = None

banner("STAGE26-G1 :: REMOTE BYTE VERIFICATION")
git("fetch", "origin", "main")
local = git("rev-parse", "HEAD")
remote2 = git("rev-parse", "origin/main")
print("Local HEAD :", local)
print("origin/main:", remote2)
if local != new_head or remote2 != new_head:
    raise RuntimeError("Local/remote commit mismatch")

for p in files_for_manifest + [manifest_json]:
    rel = str(p.relative_to(REPO))
    remote_bytes = subprocess.run(
        ["git", "show", f"origin/main:{rel}"], cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False
    )
    if remote_bytes.returncode != 0:
        raise RuntimeError(remote_bytes.stderr.decode("utf-8", errors="replace"))
    local_bytes = p.read_bytes()
    ok = local_bytes == remote_bytes.stdout
    print(f"{'PASS' if ok else 'FAIL'} {rel}")
    if not ok:
        raise RuntimeError(f"Remote byte mismatch: {rel}")

final_status = git("status", "--porcelain")

# ==================================================================================================
# 7. FINAL DISPLAY
# ==================================================================================================

banner("STAGE26-G1 COMPLETE :: CONSOLIDATED GPU PROFILE")
print("G1 durable anchor           :", new_head)
print("HEAD == origin/main         :", local == remote2 == new_head)
print("Repo clean                  :", final_status == "")
print("Physical GPU index          :", PHYSICAL_GPU_INDEX)
print("Hardware mode               : GPU_SINGLE_T4")
print("Status counts               :", status_counts)
print("Raw timing observations     :", len(raw_observations))
print("Bootstrap reps              :", BOOTSTRAP_REPS)
print("GPU memory profile          : COMPLETE FOR PASS CONDITIONS")
print("LightGBM GPU inference      : BACKEND_UNAVAILABLE")
print("LGBM+XGB ensemble GPU       : BACKEND_UNAVAILABLE")
print("Raw PCAP                     : NOT ACCESSED")
print("Compact corpus               : NOT REQUIRED")
print("Complete E2E                 : UNAVAILABLE")
print("\nNEXT:")
print("  Paste the COMPLETE output.")
print("  If G1 passes, next is a short GPU publication/CPU-vs-GPU closure cell, not another measurement campaign.")

if final_status:
    raise RuntimeError("Repository not clean after G1")



STAGE26-G1 :: DURABLE SCIENTIFIC PARENT GATE
Expected parent: 304e5613627a744cbd5d369857f8ac5667a520eb
Local HEAD     : 304e5613627a744cbd5d369857f8ac5667a520eb
origin/main    : 304e5613627a744cbd5d369857f8ac5667a520eb
Repo clean     : True

STAGE26-G1 :: PROSPECTIVE SINGLE-T4 DEVICE LOCK
1, Tesla T4, GPU-899c600e-794e-3a2e-3c0e-fccc6615ffc4, 580.159.04, 15360, 14909
Existing compute processes on selected GPU: 0
Selection rationale: physical GPU 1 was not used by G0B smoke probes; selection is contamination avoidance, not performance selection.

STAGE26-G1 :: FREEZE RANDOMIZED GPU CONDITION PLAN
Worker py_compile: PASS
Worker SHA256     : cbe469866aa0b1a33b31225e9946b25be6ae7410abf27461e0620facee3f1e2e
Conditions total       : 40
Supported measurements : 30
Backend unavailable    : 10
Randomization seed     : 26042
Selected physical GPU  : 1

STAGE26-G1 :: EXECUTE RANDOMIZED GPU CONDITIONS
[01/40] GPUCOND_021  STAGE20_MASKED_CNN_V1                          B=1
    -> PASS  timed=200  

RuntimeError: Command failed (128): git commit -m stage26: profile frozen models on single T4 GPU
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@c386205d4b1c.(none)')


In [4]:
# =============================================================================
# STAGE26-G1R2 :: RESUME DURABLE COMMIT/PUSH AFTER MISSING GIT IDENTITY
#
# IMPORTANT:
#   - DOES NOT RERUN GPU PROFILING
#   - DOES NOT MODIFY MEASURED SCIENTIFIC FILES
#   - Verifies all 7 already-generated files against the hashes printed by G1
#   - Configures repository-local Git identity from the frozen parent commit
#   - Commits, pushes, and remotely byte-verifies the G1 package
# =============================================================================

from __future__ import annotations

import hashlib
import os
import stat
import subprocess
import tempfile
from pathlib import Path


REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "304e5613627a744cbd5d369857f8ac5667a520eb"
)

OUT = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_g1_gpu_warm_profile"
)

COMMIT_MSG = (
    "stage26: profile frozen models on single T4 GPU"
)

EXPECTED_FILES = {
    "stage26_g1_gpu_execution_plan.csv":
        "3d19e36b9b6000de853e291468a46b668df4301dac4c9304a3b72dda04469273",

    "stage26_g1_gpu_raw_timing.csv":
        "c41ffa687b3b779762f0adae2808d5e6489c4402d57fd8029d1c45f601e5565c",

    "stage26_g1_gpu_condition_summary.csv":
        "b355ce66882d1ebcc0f4a1a3cd030b81f2dac25f462761d22bcd0428b3bc2625",

    "stage26_g1_gpu_worker_results.json":
        "4253b8a9992f413e8c77225f54fd7ace7078b19e6fc459ff7211251b09ce98ac",

    "stage26_g1_gpu_environment.json":
        "dde88a5ebafe82ccca466db967a4797d440bcbfa006429427d64ecce6c7b6e5f",

    "stage26_g1_gpu_profile_receipt.json":
        "4ed7c3999ef6f6fffd8361ebca04f9a4398565a00953c50d97acae922c8502c2",

    "stage26_g1_gpu_profile_manifest.json":
        "7aa8ac38b9cdc38aad43d4380f850c334f5235cac61378289c3a8295457e6048",
}


def banner(text):
    print("\n" + "=" * 122)
    print(text)
    print("=" * 122)


def run(cmd, *, env=None, check=True):
    p = subprocess.run(
        cmd,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(map(str, cmd))}\n{p.stdout}"
        )

    return p.stdout.strip()


def git(*args, env=None, check=True):
    return run(
        ["git", *args],
        env=env,
        check=check,
    )


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for block in iter(
            lambda: f.read(16 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def remote_blob(ref, rel):
    p = subprocess.run(
        [
            "git",
            "show",
            f"{ref}:{rel}",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stderr.decode(
                "utf-8",
                errors="replace",
            )
        )

    return p.stdout


# =============================================================================
# 1. RECOVERY STATE
# =============================================================================

banner("STAGE26-G1R2 :: RECOVERY STATE")

head = git(
    "rev-parse",
    "HEAD",
)

origin_main = git(
    "rev-parse",
    "origin/main",
)

print("Expected parent :", EXPECTED_PARENT)
print("Local HEAD      :", head)
print("origin/main     :", origin_main)

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "HEAD changed after the failed commit attempt. "
        "Do not continue automatically."
    )

if origin_main != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main changed unexpectedly."
    )


unstaged = git(
    "diff",
    "--name-only",
).splitlines()

staged = git(
    "diff",
    "--cached",
    "--name-only",
).splitlines()

untracked = git(
    "ls-files",
    "--others",
    "--exclude-standard",
).splitlines()


print("Unstaged tracked files :", len([x for x in unstaged if x]))
print("Staged files           :", len([x for x in staged if x]))
print("Untracked files        :", len([x for x in untracked if x]))


if unstaged:
    print("\nUnexpected unstaged tracked files:")
    print("\n".join(unstaged))
    raise RuntimeError(
        "Unexpected unstaged tracked modifications."
    )


expected_rel = sorted(
    str(
        (
            OUT
            / filename
        ).relative_to(REPO)
    )
    for filename in EXPECTED_FILES
)


if sorted(staged) != expected_rel:
    print("\nExpected staged files:")
    for x in expected_rel:
        print(" ", x)

    print("\nActual staged files:")
    for x in staged:
        print(" ", x)

    raise RuntimeError(
        "Staged file set differs from the completed G1 package."
    )


if untracked:
    print("\nUnexpected untracked files:")
    for x in untracked:
        print(" ", x)

    raise RuntimeError(
        "Unexpected untracked files after G1."
    )


print("Recovery repository geometry: PASS")


# =============================================================================
# 2. VERIFY EXISTING MEASUREMENT PACKAGE BY HASH
# =============================================================================

banner("STAGE26-G1R2 :: VERIFY EXISTING G1 PACKAGE")

for filename, expected_sha in EXPECTED_FILES.items():

    path = OUT / filename

    if not path.is_file():
        raise FileNotFoundError(
            path
        )

    actual_sha = sha256_file(
        path
    )

    print(filename)
    print("  expected:", expected_sha)
    print("  actual  :", actual_sha)
    print("  PASS    :", actual_sha == expected_sha)

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"G1 file changed after measurement: {filename}"
        )


print("\nAll 7 G1 files are byte-identical to the completed measurement run.")


# =============================================================================
# 3. CONFIRM STAGED BYTES MATCH WORKTREE BY HASH
# =============================================================================

banner("STAGE26-G1R2 :: VERIFY INDEX/STAGED BYTES")

for filename, expected_sha in EXPECTED_FILES.items():

    rel = str(
        (
            OUT
            / filename
        ).relative_to(REPO)
    )

    p = subprocess.run(
        [
            "git",
            "show",
            f":{rel}",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            f"Unable to read staged blob: {rel}\n"
            + p.stderr.decode(
                "utf-8",
                errors="replace",
            )
        )

    staged_sha = hashlib.sha256(
        p.stdout
    ).hexdigest()

    print(
        f"{filename:46s} "
        f"{'PASS' if staged_sha == expected_sha else 'FAIL'}"
    )

    if staged_sha != expected_sha:
        raise RuntimeError(
            f"Staged bytes differ for {filename}"
        )


# =============================================================================
# 4. CONFIGURE REPOSITORY-LOCAL GIT IDENTITY FROM FROZEN PARENT
# =============================================================================

banner("STAGE26-G1R2 :: CONFIGURE LOCAL GIT IDENTITY")

parent_author_name = git(
    "show",
    "-s",
    "--format=%an",
    EXPECTED_PARENT,
)

parent_author_email = git(
    "show",
    "-s",
    "--format=%ae",
    EXPECTED_PARENT,
)


if not parent_author_name.strip():
    raise RuntimeError(
        "Could not recover parent author name."
    )

if not parent_author_email.strip():
    raise RuntimeError(
        "Could not recover parent author email."
    )


git(
    "config",
    "--local",
    "user.name",
    parent_author_name,
)

git(
    "config",
    "--local",
    "user.email",
    parent_author_email,
)


configured_name = git(
    "config",
    "--local",
    "--get",
    "user.name",
)

configured_email = git(
    "config",
    "--local",
    "--get",
    "user.email",
)


print("Recovered author name :", configured_name)
print("Recovered author email:", configured_email)
print("Scope                 : REPOSITORY LOCAL ONLY")


if configured_name != parent_author_name:
    raise RuntimeError(
        "Git author-name configuration mismatch."
    )

if configured_email != parent_author_email:
    raise RuntimeError(
        "Git author-email configuration mismatch."
    )


# =============================================================================
# 5. COMMIT — NO MEASUREMENT REEXECUTION
# =============================================================================

banner("STAGE26-G1R2 :: RESUME COMMIT")

git(
    "commit",
    "-m",
    COMMIT_MSG,
)


new_head = git(
    "rev-parse",
    "HEAD",
)

new_parent = git(
    "rev-parse",
    "HEAD^",
)

subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print("Parent :", new_parent)
print("HEAD   :", new_head)
print("Subject:", subject)


if new_parent != EXPECTED_PARENT:
    raise RuntimeError(
        "G1 commit parent mismatch."
    )

if subject != COMMIT_MSG:
    raise RuntimeError(
        "G1 commit subject mismatch."
    )


# =============================================================================
# 6. PUSH USING KAGGLE SECRET
# =============================================================================

banner("STAGE26-G1R2 :: PUSH")

from kaggle_secrets import UserSecretsClient


token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if (
    not token
    or len(token.strip()) < 20
):
    raise RuntimeError(
        "GITHUB_TOKEN unavailable."
    )


fd, askpass_name = tempfile.mkstemp(
    prefix="stage26_g1r2_askpass_",
    suffix=".sh",
)

os.close(fd)

askpass = Path(
    askpass_name
)


try:

    askpass.write_text(
        "#!/bin/sh\n"
        "case \"$1\" in\n"
        "  *Username*) printf \"%s\\n\" \"x-access-token\" ;;\n"
        "  *Password*) printf \"%s\\n\" \"$STAGE26_GITHUB_TOKEN\" ;;\n"
        "  *) printf \"%s\\n\" \"\" ;;\n"
        "esac\n",
        encoding="utf-8",
    )

    askpass.chmod(
        askpass.stat().st_mode
        | stat.S_IXUSR
        | stat.S_IXGRP
        | stat.S_IXOTH
    )

    env = os.environ.copy()

    env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    env[
        "STAGE26_GITHUB_TOKEN"
    ] = token.strip()


    print(
        git(
            "push",
            "origin",
            "main",
            env=env,
        )
    )

finally:

    askpass.unlink(
        missing_ok=True
    )

    token = None

    if "env" in locals():
        env.pop(
            "STAGE26_GITHUB_TOKEN",
            None,
        )


# =============================================================================
# 7. REMOTE COMMIT + BYTE VERIFICATION
# =============================================================================

banner("STAGE26-G1R2 :: REMOTE BYTE VERIFICATION")

git(
    "fetch",
    "origin",
    "main",
)


local_head = git(
    "rev-parse",
    "HEAD",
)

remote_head = git(
    "rev-parse",
    "origin/main",
)


print("Local HEAD :", local_head)
print("origin/main:", remote_head)


if local_head != new_head:
    raise RuntimeError(
        "Local HEAD changed unexpectedly."
    )

if remote_head != new_head:
    raise RuntimeError(
        "Push did not establish G1 commit at origin/main."
    )


for filename, expected_sha in EXPECTED_FILES.items():

    path = OUT / filename

    rel = str(
        path.relative_to(
            REPO
        )
    )

    remote_bytes = remote_blob(
        "origin/main",
        rel,
    )

    remote_sha = hashlib.sha256(
        remote_bytes
    ).hexdigest()

    local_sha = sha256_file(
        path
    )


    print(filename)
    print("  expected:", expected_sha)
    print("  local   :", local_sha)
    print("  remote  :", remote_sha)
    print(
        "  PASS    :",
        (
            expected_sha
            ==
            local_sha
            ==
            remote_sha
        ),
    )


    if not (
        expected_sha
        ==
        local_sha
        ==
        remote_sha
    ):
        raise RuntimeError(
            f"Remote byte verification failed: {filename}"
        )


# =============================================================================
# 8. FINAL CLEAN STATE
# =============================================================================

banner("STAGE26-G1R2 COMPLETE")

final_status = git(
    "status",
    "--porcelain",
)


print("G1 durable anchor       :", new_head)
print(
    "HEAD == origin/main     :",
    local_head == remote_head == new_head,
)
print(
    "Repo clean              :",
    final_status == "",
)

print()
print("GPU conditions total    : 40")
print("PASS                    : 29")
print("BACKEND_UNAVAILABLE     : 10")
print("RESOURCE_LIMIT_OOM      : 1")
print("Raw timing observations : 3100")

print()
print("GPU measurement rerun   : NO")
print("Scientific files changed: NO")
print("Raw PCAP accessed       : NO")
print("Missing cost imputed    : NO")

print()
print(
    "NEXT: GPU results are durable. "
    "Proceed to GPU result audit / CPU-vs-GPU publication closure."
)


if final_status:
    raise RuntimeError(
        "Repository is not clean after G1 recovery."
    )


STAGE26-G1R2 :: RECOVERY STATE
Expected parent : 304e5613627a744cbd5d369857f8ac5667a520eb
Local HEAD      : 304e5613627a744cbd5d369857f8ac5667a520eb
origin/main     : 304e5613627a744cbd5d369857f8ac5667a520eb
Unstaged tracked files : 0
Staged files           : 7
Untracked files        : 0
Recovery repository geometry: PASS

STAGE26-G1R2 :: VERIFY EXISTING G1 PACKAGE
stage26_g1_gpu_execution_plan.csv
  expected: 3d19e36b9b6000de853e291468a46b668df4301dac4c9304a3b72dda04469273
  actual  : 3d19e36b9b6000de853e291468a46b668df4301dac4c9304a3b72dda04469273
  PASS    : True
stage26_g1_gpu_raw_timing.csv
  expected: c41ffa687b3b779762f0adae2808d5e6489c4402d57fd8029d1c45f601e5565c
  actual  : c41ffa687b3b779762f0adae2808d5e6489c4402d57fd8029d1c45f601e5565c
  PASS    : True
stage26_g1_gpu_condition_summary.csv
  expected: b355ce66882d1ebcc0f4a1a3cd030b81f2dac25f462761d22bcd0428b3bc2625
  actual  : b355ce66882d1ebcc0f4a1a3cd030b81f2dac25f462761d22bcd0428b3bc2625
  PASS    : True
stage26_g1_gpu_wo

In [5]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import stat
import subprocess
import tempfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

REPO = Path('/kaggle/working/ids2018-validation-safe-ablation')
ROOT = REPO / 'results' / 'stage26_deployment_profiling'
EXPECTED_PARENT = '79b9c23c91ef7185d35b700e6f6f84aefb54ab35'
CPU_CLOSURE_ANCHOR = '304e5613627a744cbd5d369857f8ac5667a520eb'
PROTOCOL_SHA = 'd79cddec0cc353ed0e02031134ebdf14dffbc3de73ffd1f1e1feff061d08c625'
GPU_CONTRACT_SHA = '09dfeb386f462597d4fca76aad5676d20901017c6f8175d221d2f85e108d59b7'
CPU_WARM_SHA = 'd664feca94308ea544506df7ef5c997bc4486f457153abfc1959db36808cfaad'

G1DIR = ROOT / 'stage26_g1_gpu_warm_profile'
CPU_WARM = ROOT / 'stage26_8d3_cpu_publication_tables' / 'T26_CPU_WARM_INFERENCE.csv'
OUT = ROOT / 'stage26_g2_final_gpu_publication_closure'

SOURCE_SHA = {
    'stage26_g1_gpu_execution_plan.csv': '3d19e36b9b6000de853e291468a46b668df4301dac4c9304a3b72dda04469273',
    'stage26_g1_gpu_raw_timing.csv': 'c41ffa687b3b779762f0adae2808d5e6489c4402d57fd8029d1c45f601e5565c',
    'stage26_g1_gpu_condition_summary.csv': 'b355ce66882d1ebcc0f4a1a3cd030b81f2dac25f462761d22bcd0428b3bc2625',
    'stage26_g1_gpu_worker_results.json': '4253b8a9992f413e8c77225f54fd7ace7078b19e6fc459ff7211251b09ce98ac',
    'stage26_g1_gpu_environment.json': 'dde88a5ebafe82ccca466db967a4797d440bcbfa006429427d64ecce6c7b6e5f',
    'stage26_g1_gpu_profile_receipt.json': '4ed7c3999ef6f6fffd8361ebca04f9a4398565a00953c50d97acae922c8502c2',
    'stage26_g1_gpu_profile_manifest.json': '7aa8ac38b9cdc38aad43d4380f850c334f5235cac61378289c3a8295457e6048',
}

TARGET_ORDER = [
    'STAGE16_XGBOOST_TUNED',
    'STAGE16_LIGHTGBM_TUNED',
    'STAGE16_CATBOOST_TUNED',
    'FT_BALANCED_5_CHECKPOINT_SOFT_VOTING',
    'STAGE20_MASKED_CNN_V1',
    'STAGE21_MASKED_VIT_V1',
    'ENS_LGBM_XGB_EQUAL',
    'FT_BALANCED_SINGLE_RESOURCE_REFERENCE',
]
BATCHES = [1, 64, 256, 1024, 8192]
GROUP_A = 'GROUP_A_DUPSAFE70'
GROUP_B = 'GROUP_B_PACKET_IMAGE'

SHORT = {
    'STAGE16_XGBOOST_TUNED': 'XGBoost',
    'STAGE16_LIGHTGBM_TUNED': 'LightGBM',
    'STAGE16_CATBOOST_TUNED': 'CatBoost',
    'FT_BALANCED_5_CHECKPOINT_SOFT_VOTING': 'FT ensemble',
    'STAGE20_MASKED_CNN_V1': 'CNN',
    'STAGE21_MASKED_VIT_V1': 'ViT',
    'ENS_LGBM_XGB_EQUAL': 'LGBM+XGB ensemble',
    'FT_BALANCED_SINGLE_RESOURCE_REFERENCE': 'FT single',
}

COMMIT_MSG = 'stage26: close GPU profiling and CPU GPU comparison'


def banner(text):
    print('\n' + '=' * 124)
    print(text)
    print('=' * 124)


def run(cmd, *, cwd=REPO, env=None, check=True):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        check=False,
    )
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}\n{p.stdout}")
    return p.stdout.strip()


def git(*args, env=None, check=True):
    return run(['git', *args], env=env, check=check)


def sha256_file(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()


def atomic_json(path, obj):
    tmp = Path(str(path) + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, sort_keys=True, allow_nan=False)
        f.write('\n')
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def remote_blob(ref, rel):
    p = subprocess.run(
        ['git', 'show', f'{ref}:{rel}'],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )
    if p.returncode != 0:
        raise RuntimeError(p.stderr.decode('utf-8', errors='replace'))
    return p.stdout


def is_blank(v):
    return pd.isna(v) or str(v).strip() == ''


# ================================================================================================
# 1. DURABLE PARENT GATE
# ================================================================================================

banner('STAGE26-G2 :: DURABLE PARENT GATE')

git('fetch', 'origin', 'main')
head = git('rev-parse', 'HEAD')
remote = git('rev-parse', 'origin/main')
status = git('status', '--porcelain')

print('Expected G1 parent:', EXPECTED_PARENT)
print('Local HEAD        :', head)
print('origin/main       :', remote)
print('Repo clean        :', status == '')

if head != EXPECTED_PARENT or remote != EXPECTED_PARENT or status:
    raise RuntimeError('G2 durable parent gate failed.')
if OUT.exists():
    raise RuntimeError(f'Output directory already exists: {OUT}')


# ================================================================================================
# 2. BYTE-VERIFY FROZEN GPU PACKAGE + CPU SOURCE
# ================================================================================================

banner('STAGE26-G2 :: SOURCE BYTE IDENTITY')

for name, expected in SOURCE_SHA.items():
    p = G1DIR / name
    actual = sha256_file(p)
    print(f'{name}\n  expected={expected}\n  actual  ={actual}')
    if actual != expected:
        raise RuntimeError(f'G1 source hash mismatch: {name}')

cpu_warm_actual = sha256_file(CPU_WARM)
print(f'{CPU_WARM.name}\n  expected={CPU_WARM_SHA}\n  actual  ={cpu_warm_actual}')
if cpu_warm_actual != CPU_WARM_SHA:
    raise RuntimeError('Frozen CPU warm table hash mismatch.')


# ================================================================================================
# 3. AUDIT G1 SCIENTIFIC RECEIPT / GEOMETRY
# ================================================================================================

banner('STAGE26-G2 :: GPU RESULT AUDIT')

receipt = json.loads((G1DIR / 'stage26_g1_gpu_profile_receipt.json').read_text(encoding='utf-8'))
envrec = json.loads((G1DIR / 'stage26_g1_gpu_environment.json').read_text(encoding='utf-8'))
gpu = pd.read_csv(G1DIR / 'stage26_g1_gpu_condition_summary.csv')
raw = pd.read_csv(G1DIR / 'stage26_g1_gpu_raw_timing.csv')
cpu = pd.read_csv(CPU_WARM)

if receipt['scientific_parent'] != CPU_CLOSURE_ANCHOR:
    raise RuntimeError('G1 scientific parent mismatch.')
if receipt['condition_count'] != 40 or receipt['supported_condition_count'] != 30:
    raise RuntimeError('G1 condition geometry mismatch.')
if receipt['raw_timing_observation_count'] != 3100 or len(raw) != 3100:
    raise RuntimeError('G1 raw timing count mismatch.')
if receipt['status_counts'] != {'BACKEND_UNAVAILABLE': 10, 'PASS': 29, 'RESOURCE_LIMIT_OOM': 1}:
    raise RuntimeError(f"G1 status counts changed: {receipt['status_counts']}")
if receipt['timing']['gpu_synchronization'] != 'DEVICE_SYNCHRONIZE_BEFORE_AND_AFTER_EVERY_TIMED_REGION':
    raise RuntimeError('GPU synchronization policy mismatch.')
if receipt['timing']['boundary'] != 'PREPARED_MODEL_INPUT_TO_HOST_NUMPY_ATTACK_PROBABILITY':
    raise RuntimeError('GPU timing boundary mismatch.')
if receipt['memory']['timing_and_memory_runs_separate'] is not True:
    raise RuntimeError('GPU timing/memory separation violated.')
if receipt['scientific_boundaries']['raw_pcap_accessed'] is not False:
    raise RuntimeError('Raw PCAP access unexpectedly reported.')
if receipt['scientific_boundaries']['complete_e2e_measurement'] != 'UNAVAILABLE':
    raise RuntimeError('Complete E2E boundary changed.')
if receipt['backend_policy']['no_retraining'] is not True or receipt['backend_policy']['no_model_conversion'] is not True:
    raise RuntimeError('Frozen model policy violated.')
if envrec['measurement_protocol_sha256'] != PROTOCOL_SHA:
    raise RuntimeError('G1 measurement protocol SHA mismatch.')
if envrec['gpu_bootstrap_contract_sha256'] != GPU_CONTRACT_SHA:
    raise RuntimeError('GPU bootstrap contract SHA mismatch.')
if envrec['selected_physical_gpu_index'] != 1:
    raise RuntimeError('Unexpected selected physical GPU index.')

# Exact 8 x 5 geometry.
keys = set(zip(gpu['target_id'], gpu['batch_size'].astype(int)))
expected_keys = {(t, b) for t in TARGET_ORDER for b in BATCHES}
if keys != expected_keys or len(gpu) != 40:
    raise RuntimeError('GPU summary does not preserve frozen 8x5 geometry.')

counts = Counter(gpu['status'])
if counts != Counter({'PASS': 29, 'BACKEND_UNAVAILABLE': 10, 'RESOURCE_LIMIT_OOM': 1}):
    raise RuntimeError(f'GPU status geometry changed: {counts}')

# Exact backend-unavailable set.
bu = gpu[gpu['status'] == 'BACKEND_UNAVAILABLE']
if set(bu['target_id']) != {'STAGE16_LIGHTGBM_TUNED', 'ENS_LGBM_XGB_EQUAL'} or len(bu) != 10:
    raise RuntimeError('BACKEND_UNAVAILABLE set changed.')

# Exact resource-limit set.
oom = gpu[gpu['status'] == 'RESOURCE_LIMIT_OOM']
if len(oom) != 1:
    raise RuntimeError('Expected exactly one GPU OOM condition.')
oom_row = oom.iloc[0]
if oom_row['target_id'] != 'STAGE20_MASKED_CNN_V1' or int(oom_row['batch_size']) != 8192:
    raise RuntimeError('Unexpected GPU OOM condition.')

quant_cols = [
    'p50_latency_ms','p50_ci95_low_ms','p50_ci95_high_ms',
    'p95_latency_ms','p95_ci95_low_ms','p95_ci95_high_ms',
    'p99_latency_ms_if_n_gte_100','p99_ci95_low_ms_if_n_gte_100','p99_ci95_high_ms_if_n_gte_100',
    'median_throughput_flows_per_second','throughput_ci95_low_flows_per_second','throughput_ci95_high_flows_per_second',
    'gpu_memory_baseline_mib','gpu_memory_loaded_mib','gpu_memory_prepared_input_mib','gpu_memory_peak_process_mib',
    'delta_model_gpu_memory_mib','delta_peak_gpu_memory_mib','torch_peak_allocated_mib',
]
for _, r in gpu[gpu['status'] != 'PASS'].iterrows():
    if int(r['n']) != 0:
        raise RuntimeError('Non-PASS GPU condition has n != 0.')
    if any(not is_blank(r[c]) for c in quant_cols):
        raise RuntimeError(f"Non-PASS GPU cost imputation detected: {r['condition_id']}")

# Raw timing counts must exactly match per-condition n.
raw_counts = raw.groupby('condition_id').size().to_dict()
for _, r in gpu.iterrows():
    n = int(r['n'])
    observed = int(raw_counts.get(r['condition_id'], 0))
    if observed != n:
        raise RuntimeError(f"Raw timing count mismatch {r['condition_id']}: {observed} != {n}")

# p99 eligibility invariant.
for _, r in gpu[gpu['status'] == 'PASS'].iterrows():
    n = int(r['n'])
    p99_fields = [
        r['p99_latency_ms_if_n_gte_100'],
        r['p99_ci95_low_ms_if_n_gte_100'],
        r['p99_ci95_high_ms_if_n_gte_100'],
    ]
    if n >= 100 and any(is_blank(v) for v in p99_fields):
        raise RuntimeError(f"Missing eligible p99 values: {r['condition_id']}")
    if n < 100 and any(not is_blank(v) for v in p99_fields):
        raise RuntimeError(f"Ineligible p99 values populated: {r['condition_id']}")

print('40-row frozen GPU geometry       : PASS')
print('29 PASS / 10 backend / 1 OOM    : PASS')
print('3100 raw timing observations     : PASS')
print('Non-PASS cost imputation         : NONE')
print('GPU synchronization              : VERIFIED')
print('Timing/memory separation         : VERIFIED')
print('Raw PCAP                         : NOT ACCESSED')
print('Retraining/model conversion      : NONE')


# ================================================================================================
# 4. BUILD PUBLICATION TABLES — NO NEW SCIENCE
# ================================================================================================

banner('STAGE26-G2 :: BUILD PUBLICATION TABLES')

OUT.mkdir(parents=True, exist_ok=False)

# T1: GPU warm inference + memory, copied from durable G1 summary with publication labels.
gpu_pub = gpu.copy()
gpu_pub.insert(0, 'publication_scope', 'GPU_SINGLE_T4_COMPONENT_LEVEL_WARM_INFERENCE')
gpu_pub['complete_e2e_available'] = False
gpu_pub['ratio_uncertainty_available'] = False
gpu_pub['publication_claim_boundary'] = 'COMPONENT_LEVEL_ONLY'
T_GPU = OUT / 'T26_GPU_WARM_INFERENCE_MEMORY.csv'
gpu_pub.to_csv(T_GPU, index=False)

# T2: matched CPU1/GPU PASS comparison only. No post-hoc best-batch selection and no ratio CIs.
cpu1 = cpu[cpu['cpu_mode'] == 'CPU_1_PHYSICAL_CORE'].copy()
cpuk = cpu1[['target_id','batch_size','status','p50_latency_ms','p50_ci95_low_ms','p50_ci95_high_ms',
             'p95_latency_ms','p95_ci95_low_ms','p95_ci95_high_ms',
             'median_throughput_samples_per_s','throughput_ci95_low_samples_per_s','throughput_ci95_high_samples_per_s']].copy()
cpuk = cpuk.rename(columns={
    'status':'cpu_status',
    'p50_latency_ms':'cpu_p50_latency_ms',
    'p50_ci95_low_ms':'cpu_p50_ci95_low_ms',
    'p50_ci95_high_ms':'cpu_p50_ci95_high_ms',
    'p95_latency_ms':'cpu_p95_latency_ms',
    'p95_ci95_low_ms':'cpu_p95_ci95_low_ms',
    'p95_ci95_high_ms':'cpu_p95_ci95_high_ms',
    'median_throughput_samples_per_s':'cpu_median_throughput_samples_per_s',
    'throughput_ci95_low_samples_per_s':'cpu_throughput_ci95_low_samples_per_s',
    'throughput_ci95_high_samples_per_s':'cpu_throughput_ci95_high_samples_per_s',
})

gpuk = gpu[['target_id','comparison_group','batch_size','status','p50_latency_ms','p50_ci95_low_ms','p50_ci95_high_ms',
            'p95_latency_ms','p95_ci95_low_ms','p95_ci95_high_ms',
            'median_throughput_flows_per_second','throughput_ci95_low_flows_per_second','throughput_ci95_high_flows_per_second']].copy()
gpuk = gpuk.rename(columns={
    'status':'gpu_status',
    'p50_latency_ms':'gpu_p50_latency_ms',
    'p50_ci95_low_ms':'gpu_p50_ci95_low_ms',
    'p50_ci95_high_ms':'gpu_p50_ci95_high_ms',
    'p95_latency_ms':'gpu_p95_latency_ms',
    'p95_ci95_low_ms':'gpu_p95_ci95_low_ms',
    'p95_ci95_high_ms':'gpu_p95_ci95_high_ms',
    'median_throughput_flows_per_second':'gpu_median_throughput_samples_per_s',
    'throughput_ci95_low_flows_per_second':'gpu_throughput_ci95_low_samples_per_s',
    'throughput_ci95_high_flows_per_second':'gpu_throughput_ci95_high_samples_per_s',
})

matched = gpuk.merge(cpuk, on=['target_id','batch_size'], how='inner', validate='one_to_one')
matched = matched[(matched['gpu_status'] == 'PASS') & (matched['cpu_status'] == 'PASS')].copy()
matched['p50_latency_speedup_cpu_over_gpu'] = matched['cpu_p50_latency_ms'] / matched['gpu_p50_latency_ms']
matched['p95_latency_speedup_cpu_over_gpu'] = matched['cpu_p95_latency_ms'] / matched['gpu_p95_latency_ms']
matched['throughput_ratio_gpu_over_cpu'] = matched['gpu_median_throughput_samples_per_s'] / matched['cpu_median_throughput_samples_per_s']
matched['ratio_uncertainty_available'] = False
matched['ratio_interpretation'] = 'POINT_ESTIMATE_ONLY__GT1_GPU_ADVANTAGE__LT1_CPU_ADVANTAGE'
matched['comparison_boundary'] = 'MATCHED_PREPARED_INPUT_TO_MATERIALIZED_PROBABILITY_COMPONENT_LEVEL_ONLY'
matched = matched.sort_values(['comparison_group','target_id','batch_size']).reset_index(drop=True)

if len(matched) != 26:
    raise RuntimeError(f'Expected 26 matched CPU1/GPU PASS rows, got {len(matched)}')
T_MATCH = OUT / 'T26_CPU1_GPU_MATCHED_COMPARISON.csv'
matched.to_csv(T_MATCH, index=False)

# T3: backend/resource availability by target.
status_rows = []
for target in TARGET_ORDER:
    d = gpu[gpu['target_id'] == target].copy()
    c = Counter(d['status'])
    if c['BACKEND_UNAVAILABLE'] == 5:
        availability = 'BACKEND_UNAVAILABLE'
    elif c['PASS'] == 5:
        availability = 'GPU_NATIVE_MEASURED_ALL_BATCHES'
    elif c['PASS'] > 0 and c['RESOURCE_LIMIT_OOM'] > 0:
        availability = 'GPU_NATIVE_MEASURED_WITH_RESOURCE_LIMIT'
    else:
        availability = 'MIXED_OR_UNEXPECTED'
    status_rows.append({
        'target_id': target,
        'comparison_group': d.iloc[0]['comparison_group'],
        'gpu_backend_availability': availability,
        'pass_conditions': int(c['PASS']),
        'backend_unavailable_conditions': int(c['BACKEND_UNAVAILABLE']),
        'resource_limit_oom_conditions': int(c['RESOURCE_LIMIT_OOM']),
        'measured_batches': ';'.join(map(str, sorted(d.loc[d['status']=='PASS','batch_size'].astype(int).tolist()))),
        'backend_unavailable_batches': ';'.join(map(str, sorted(d.loc[d['status']=='BACKEND_UNAVAILABLE','batch_size'].astype(int).tolist()))),
        'resource_limit_batches': ';'.join(map(str, sorted(d.loc[d['status']=='RESOURCE_LIMIT_OOM','batch_size'].astype(int).tolist()))),
        'missing_cost_imputed': False,
    })
backend = pd.DataFrame(status_rows)
T_STATUS = OUT / 'T26_GPU_BACKEND_RESOURCE_STATUS.csv'
backend.to_csv(T_STATUS, index=False)

# T4: Batch-1 publication anchor, point estimates only for ratio columns.
b1 = matched[matched['batch_size'] == 1].copy()
T_B1 = OUT / 'T26_CPU1_GPU_BATCH1_ANCHOR.csv'
b1.to_csv(T_B1, index=False)

print('GPU warm/memory rows      :', len(gpu_pub))
print('Matched CPU1/GPU rows     :', len(matched))
print('Backend status rows       :', len(backend))
print('Matched B1 anchor rows    :', len(b1))


# ================================================================================================
# 5. FIGURES — DESCRIPTIVE ONLY
# ================================================================================================

banner('STAGE26-G2 :: RENDER DESCRIPTIVE PUBLICATION FIGURES')

# Figure 1: p95 speedup CPU/GPU, groups separate, no ratio CI, no best-batch selection.
fig, axes = plt.subplots(1, 2, figsize=(14, 5.6), constrained_layout=True)
for ax, group, title in zip(axes, [GROUP_A, GROUP_B], ['Group A: duplicate-safe 70-feature', 'Group B: packet-image']):
    sub = matched[matched['comparison_group'] == group]
    for target in TARGET_ORDER:
        d = sub[sub['target_id'] == target]
        if d.empty:
            continue
        xs = [BATCHES.index(int(b)) for b in d['batch_size']]
        ys = d['p95_latency_speedup_cpu_over_gpu'].to_numpy(float)
        ax.scatter(xs, ys, label=SHORT[target], s=38)
    ax.axhline(1.0, linewidth=1.0, linestyle='--')
    ax.set_xticks(range(len(BATCHES)), [str(x) for x in BATCHES])
    ax.set_yscale('log')
    ax.set_xlabel('Frozen batch size')
    ax.set_ylabel('CPU1 p95 / GPU p95 (point estimate)')
    ax.set_title(title)
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=8)
fig.suptitle('Stage26 matched CPU1 vs single-T4 p95 latency speedup\nDescriptive point estimates; no ratio confidence intervals', fontsize=12)
F_SPEED_PNG = OUT / 'F26_CPU1_GPU_P95_SPEEDUP.png'
F_SPEED_PDF = OUT / 'F26_CPU1_GPU_P95_SPEEDUP.pdf'
fig.savefig(F_SPEED_PNG, dpi=300, metadata={'Software':'Stage26-G2'})
fig.savefig(F_SPEED_PDF, metadata={'Creator':'Stage26-G2','CreationDate':None,'ModDate':None})
plt.close(fig)

# Figure 2: GPU delta peak process memory by target x batch; non-PASS cells not imputed.
fig, ax = plt.subplots(figsize=(12.5, 6.0), constrained_layout=True)
arr = np.full((len(TARGET_ORDER), len(BATCHES)), np.nan, dtype=float)
labels = [['' for _ in BATCHES] for _ in TARGET_ORDER]
for i, target in enumerate(TARGET_ORDER):
    d = gpu[gpu['target_id'] == target]
    for j, batch in enumerate(BATCHES):
        r = d[d['batch_size'].astype(int) == batch].iloc[0]
        if r['status'] == 'PASS':
            arr[i, j] = float(r['delta_peak_gpu_memory_mib'])
            labels[i][j] = f"{arr[i,j]:.0f}"
        elif r['status'] == 'RESOURCE_LIMIT_OOM':
            labels[i][j] = 'OOM'
        elif r['status'] == 'BACKEND_UNAVAILABLE':
            labels[i][j] = 'N/A'

masked = np.ma.masked_invalid(arr)
im = ax.imshow(masked, aspect='auto')
ax.set_xticks(range(len(BATCHES)), [str(x) for x in BATCHES])
ax.set_yticks(range(len(TARGET_ORDER)), [SHORT[x] for x in TARGET_ORDER])
ax.set_xlabel('Frozen batch size')
ax.set_title('Single-T4 delta peak process GPU memory (MiB)\nNon-PASS outcomes retained; no imputation')
for i in range(len(TARGET_ORDER)):
    for j in range(len(BATCHES)):
        if labels[i][j]:
            ax.text(j, i, labels[i][j], ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, label='Delta peak GPU memory (MiB)')
F_MEM_PNG = OUT / 'F26_GPU_DELTA_PEAK_MEMORY.png'
F_MEM_PDF = OUT / 'F26_GPU_DELTA_PEAK_MEMORY.pdf'
fig.savefig(F_MEM_PNG, dpi=300, metadata={'Software':'Stage26-G2'})
fig.savefig(F_MEM_PDF, metadata={'Creator':'Stage26-G2','CreationDate':None,'ModDate':None})
plt.close(fig)

print('Rendered figure families: 2')
print('Rendered files          : 4')


# ================================================================================================
# 6. PUBLICATION INDEX / RECEIPT / MANIFEST
# ================================================================================================

banner('STAGE26-G2 :: WRITE FINAL STAGE26 CLOSURE METADATA')

now = datetime.now(timezone.utc).isoformat()

# Useful B1 point estimates for publication narrative, source values only.
b1_summary = []
for _, r in b1.sort_values(['comparison_group','target_id']).iterrows():
    b1_summary.append({
        'target_id': r['target_id'],
        'comparison_group': r['comparison_group'],
        'cpu1_p95_latency_ms': float(r['cpu_p95_latency_ms']),
        'gpu_p95_latency_ms': float(r['gpu_p95_latency_ms']),
        'p95_latency_speedup_cpu_over_gpu': float(r['p95_latency_speedup_cpu_over_gpu']),
        'cpu1_median_throughput_samples_per_s': float(r['cpu_median_throughput_samples_per_s']),
        'gpu_median_throughput_samples_per_s': float(r['gpu_median_throughput_samples_per_s']),
        'throughput_ratio_gpu_over_cpu': float(r['throughput_ratio_gpu_over_cpu']),
    })

index = {
    'schema': 'stage26_g2_final_gpu_publication_closure_index_v1',
    'created_at_utc': now,
    'scientific_parent': EXPECTED_PARENT,
    'cpu_closure_anchor': CPU_CLOSURE_ANCHOR,
    'measurement_protocol_sha256': PROTOCOL_SHA,
    'gpu_bootstrap_contract_sha256': GPU_CONTRACT_SHA,
    'source_policy': 'DURABLE_CPU_8D3_TABLE_PLUS_DURABLE_GPU_G1_PACKAGE_ONLY',
    'new_measurement_executed': False,
    'tables': [
        {'id':'T26_GPU_WARM_INFERENCE_MEMORY','path':str(T_GPU.relative_to(REPO)),'rows':int(len(gpu_pub))},
        {'id':'T26_CPU1_GPU_MATCHED_COMPARISON','path':str(T_MATCH.relative_to(REPO)),'rows':int(len(matched))},
        {'id':'T26_GPU_BACKEND_RESOURCE_STATUS','path':str(T_STATUS.relative_to(REPO)),'rows':int(len(backend))},
        {'id':'T26_CPU1_GPU_BATCH1_ANCHOR','path':str(T_B1.relative_to(REPO)),'rows':int(len(b1))},
    ],
    'figures': [
        {
            'id':'F26_CPU1_GPU_P95_SPEEDUP',
            'files':[str(F_SPEED_PNG.relative_to(REPO)), str(F_SPEED_PDF.relative_to(REPO))],
            'claim_boundary':'DESCRIPTIVE_POINT_ESTIMATE_SPEEDUP_ONLY__NO_RATIO_CI__GROUPS_SEPARATE',
        },
        {
            'id':'F26_GPU_DELTA_PEAK_MEMORY',
            'files':[str(F_MEM_PNG.relative_to(REPO)), str(F_MEM_PDF.relative_to(REPO))],
            'claim_boundary':'GPU_PROCESS_MEMORY_COMPONENT_ONLY__NONPASS_NOT_IMPUTED',
        },
    ],
    'gpu_status_counts': dict(counts),
    'matched_cpu1_gpu_pass_rows': int(len(matched)),
    'batch1_point_estimates': b1_summary,
}
INDEX = OUT / 'stage26_g2_final_gpu_publication_index.json'
atomic_json(INDEX, index)

receipt_out = {
    'schema': 'stage26_g2_final_gpu_publication_closure_receipt_v1',
    'created_at_utc': now,
    'scientific_parent': EXPECTED_PARENT,
    'cpu_closure_anchor': CPU_CLOSURE_ANCHOR,
    'measurement_protocol_sha256': PROTOCOL_SHA,
    'gpu_bootstrap_contract_sha256': GPU_CONTRACT_SHA,
    'gpu_measurement_status': 'COMPLETE_AND_DURABLE',
    'gpu_status_counts': {'PASS':29,'BACKEND_UNAVAILABLE':10,'RESOURCE_LIMIT_OOM':1},
    'raw_timing_observations': 3100,
    'matched_cpu1_gpu_pass_rows': 26,
    'scientific_boundaries': {
        'complete_e2e_measurement': 'UNAVAILABLE',
        'cross_group_pareto': 'PROHIBITED',
        'cpu_gpu_speedup_ratio_ci': 'NOT_COMPUTED',
        'post_hoc_best_batch_selection': 'PROHIBITED',
        'missing_cost_imputation': 'PROHIBITED',
        'raw_pcap_accessed': False,
        'compact_corpus_accessed': False,
        'retraining': False,
        'model_conversion': False,
        'new_timing': False,
        'new_gpu_inference': False,
        'new_gpu_memory_measurement': False,
    },
    'backend_availability': {
        'STAGE16_LIGHTGBM_TUNED': 'BACKEND_UNAVAILABLE',
        'ENS_LGBM_XGB_EQUAL': 'BACKEND_UNAVAILABLE_DUE_TO_REQUIRED_LIGHTGBM_COMPONENT',
        'STAGE16_XGBOOST_TUNED': 'GPU_NATIVE_MEASURED',
        'STAGE16_CATBOOST_TUNED': 'GPU_NATIVE_MEASURED',
        'FT_BALANCED_5_CHECKPOINT_SOFT_VOTING': 'GPU_NATIVE_MEASURED',
        'FT_BALANCED_SINGLE_RESOURCE_REFERENCE': 'GPU_NATIVE_MEASURED',
        'STAGE20_MASKED_CNN_V1': 'GPU_NATIVE_MEASURED_WITH_B8192_RESOURCE_LIMIT_OOM',
        'STAGE21_MASKED_VIT_V1': 'GPU_NATIVE_MEASURED',
    },
    'stage26_status': 'COMPLETE',
    'gpu_phase_status': 'COMPLETE',
    'publication_package_status': 'CPU_AND_GPU_COMPONENT_LEVEL_PROFILE_READY',
}
RECEIPT = OUT / 'stage26_g2_final_stage26_closure_receipt.json'
atomic_json(RECEIPT, receipt_out)

payload_files = [
    T_GPU, T_MATCH, T_STATUS, T_B1,
    F_SPEED_PNG, F_SPEED_PDF, F_MEM_PNG, F_MEM_PDF,
    INDEX, RECEIPT,
]
manifest = {
    'schema': 'stage26_g2_final_gpu_publication_closure_manifest_v1',
    'created_at_utc': now,
    'scientific_parent': EXPECTED_PARENT,
    'files': [
        {
            'path': str(p.relative_to(REPO)),
            'sha256': sha256_file(p),
            'size_bytes': p.stat().st_size,
        }
        for p in payload_files
    ],
}
MANIFEST = OUT / 'stage26_g2_final_gpu_publication_closure_manifest.json'
atomic_json(MANIFEST, manifest)

all_outputs = payload_files + [MANIFEST]
for p in all_outputs:
    print(f'{p.name:52s} {p.stat().st_size:10d}  {sha256_file(p)}')


# ================================================================================================
# 7. FINAL SAFETY AUDIT OF GENERATED PACKAGE
# ================================================================================================

banner('STAGE26-G2 :: FINAL SCIENTIFIC SAFETY AUDIT')

# No ratio confidence-interval columns.
for path in [T_MATCH, T_B1]:
    hdr = pd.read_csv(path, nrows=0).columns.tolist()
    ratio_ci = [c for c in hdr if ('ratio' in c.lower() or 'speedup' in c.lower()) and 'ci' in c.lower()]
    if ratio_ci:
        raise RuntimeError(f'Ratio CI column unexpectedly generated in {path.name}: {ratio_ci}')

# No cross-group construction: figure uses separate axes; table carries group for every row.
if matched['comparison_group'].isna().any():
    raise RuntimeError('Matched comparison lost group identity.')

# Source CIs are retained, but ratios are descriptive only.
if not (matched['ratio_uncertainty_available'] == False).all():
    raise RuntimeError('Ratio uncertainty status changed.')

print('GPU result audit                     : PASS')
print('Matched CPU1/GPU comparison           : 26 PASS rows only')
print('Ratio confidence intervals            : NOT COMPUTED')
print('Post-hoc best batch                   : NONE')
print('Cross-group Pareto                    : NONE')
print('Missing deployment cost imputation    : NONE')
print('Complete E2E                          : UNAVAILABLE')
print('New timing/inference/memory measurement: NONE')


# ================================================================================================
# 8. COMMIT / PUSH / REMOTE BYTE VERIFY
# ================================================================================================

banner('STAGE26-G2 :: COMMIT')

pre = git('status', '--porcelain').splitlines()
for x in pre:
    print(x)
expected_prefix = '?? ' + str(OUT.relative_to(REPO))
if not pre or any(not x.startswith(expected_prefix) for x in pre):
    raise RuntimeError('Unexpected pre-commit repository state.')

git('add', str(OUT.relative_to(REPO)))
staged = git('diff', '--cached', '--name-only').splitlines()
expected_staged = sorted(str(p.relative_to(REPO)) for p in all_outputs)
print('Staged files:', len(staged))
for x in staged:
    print(' ', x)
if sorted(staged) != expected_staged:
    raise RuntimeError('Unexpected staged file set.')

# G1R2 already configured repository-local Git identity.
if not git('config', '--local', '--get', 'user.name', check=False):
    parent_name = git('show', '-s', '--format=%an', EXPECTED_PARENT)
    parent_email = git('show', '-s', '--format=%ae', EXPECTED_PARENT)
    git('config', '--local', 'user.name', parent_name)
    git('config', '--local', 'user.email', parent_email)

git('commit', '-m', COMMIT_MSG)
new_head = git('rev-parse', 'HEAD')
parent = git('rev-parse', 'HEAD^')
subject = git('log', '-1', '--pretty=%s')
print('Parent :', parent)
print('HEAD   :', new_head)
print('Subject:', subject)
if parent != EXPECTED_PARENT or subject != COMMIT_MSG:
    raise RuntimeError('G2 commit identity failed.')

banner('STAGE26-G2 :: PUSH')
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
if not token or len(token.strip()) < 20:
    raise RuntimeError('GITHUB_TOKEN unavailable.')
fd, askpass_name = tempfile.mkstemp(prefix='stage26_g2_askpass_', suffix='.sh')
os.close(fd)
askpass = Path(askpass_name)
try:
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *Password*) printf "%s\\n" "$STAGE26_GITHUB_TOKEN" ;;\n'
        '  *) printf "%s\\n" "" ;;\n'
        'esac\n',
        encoding='utf-8',
    )
    askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    env = os.environ.copy()
    env['GIT_ASKPASS'] = str(askpass)
    env['GIT_TERMINAL_PROMPT'] = '0'
    env['STAGE26_GITHUB_TOKEN'] = token.strip()
    print(git('push', 'origin', 'main', env=env))
finally:
    askpass.unlink(missing_ok=True)
    token = None
    if 'env' in locals():
        env.pop('STAGE26_GITHUB_TOKEN', None)

banner('STAGE26-G2 :: REMOTE BYTE VERIFICATION')
git('fetch', 'origin', 'main')
local_head = git('rev-parse', 'HEAD')
remote_head = git('rev-parse', 'origin/main')
print('Local HEAD :', local_head)
print('origin/main:', remote_head)
if local_head != new_head or remote_head != new_head:
    raise RuntimeError('Local/remote commit mismatch after G2 push.')

for p in all_outputs:
    rel = str(p.relative_to(REPO))
    lb = p.read_bytes()
    rb = remote_blob('origin/main', rel)
    lsha = hashlib.sha256(lb).hexdigest()
    rsha = hashlib.sha256(rb).hexdigest()
    ok = lb == rb
    print(f"{'PASS' if ok else 'FAIL'} {rel}")
    print('  local :', lsha)
    print('  remote:', rsha)
    if not ok:
        raise RuntimeError(f'Remote byte mismatch: {rel}')

final = git('status', '--porcelain')

banner('STAGE26-G2 COMPLETE :: STAGE26 FINAL CLOSURE')
print('FINAL_STAGE26_ANCHOR             :', new_head)
print('HEAD == origin/main              :', local_head == remote_head == new_head)
print('Repo clean                       :', final == '')
print()
print('CPU phase                        : COMPLETE / FROZEN')
print('GPU phase                        : COMPLETE / FROZEN')
print('GPU measured conditions          : 29 PASS')
print('GPU backend unavailable          : 10')
print('GPU resource-limit OOM           : 1 (CNN B8192)')
print('GPU raw timing observations      : 3100')
print('Matched CPU1/GPU PASS comparisons:', len(matched))
print()
print('Complete E2E                     : UNAVAILABLE')
print('Cross-group Pareto               : PROHIBITED')
print('CPU/GPU ratio CIs                : NOT COMPUTED')
print('Post-hoc best batch              : NOT USED')
print('Missing cost imputation          : NONE')
print('New G2 measurement               : NONE')
print()
print('STAGE26_STATUS                   : COMPLETE')
print('PUBLICATION_PROFILE_STATUS       : CPU_AND_GPU_COMPONENT_LEVEL_PROFILE_READY')
print()
print('GPU ACCELERATOR CAN NOW BE TURNED OFF.')

if final:
    raise RuntimeError('Repository is not clean after final Stage26 closure.')



STAGE26-G2 :: DURABLE PARENT GATE
Expected G1 parent: 79b9c23c91ef7185d35b700e6f6f84aefb54ab35
Local HEAD        : 79b9c23c91ef7185d35b700e6f6f84aefb54ab35
origin/main       : 79b9c23c91ef7185d35b700e6f6f84aefb54ab35
Repo clean        : True

STAGE26-G2 :: SOURCE BYTE IDENTITY
stage26_g1_gpu_execution_plan.csv
  expected=3d19e36b9b6000de853e291468a46b668df4301dac4c9304a3b72dda04469273
  actual  =3d19e36b9b6000de853e291468a46b668df4301dac4c9304a3b72dda04469273
stage26_g1_gpu_raw_timing.csv
  expected=c41ffa687b3b779762f0adae2808d5e6489c4402d57fd8029d1c45f601e5565c
  actual  =c41ffa687b3b779762f0adae2808d5e6489c4402d57fd8029d1c45f601e5565c
stage26_g1_gpu_condition_summary.csv
  expected=b355ce66882d1ebcc0f4a1a3cd030b81f2dac25f462761d22bcd0428b3bc2625
  actual  =b355ce66882d1ebcc0f4a1a3cd030b81f2dac25f462761d22bcd0428b3bc2625
stage26_g1_gpu_worker_results.json
  expected=4253b8a9992f413e8c77225f54fd7ace7078b19e6fc459ff7211251b09ce98ac
  actual  =4253b8a9992f413e8c77225f54fd7ace7078b19e6f

In [3]:
# =============================================================================
# STAGE26-R0 :: POST-STAGE26 CPU WORKSPACE REBUILD
#
# PURPOSE
#   - Rebuild fresh Kaggle CPU session after GPU shutdown/reset
#   - Clone exact FINAL Stage26 commit
#   - Verify final CPU+GPU publication closure byte-for-byte
#   - Verify critical model/result artifacts remain present
#   - Configure local Git identity
#   - Keep repo clean
#
# IMPORTANT
#   - NO new measurements
#   - NO inference
#   - NO GPU required
#   - NO raw PCAP
#   - NO compact-corpus reconstruction
#   - NO retraining
# =============================================================================

from __future__ import annotations

import hashlib
import json
import os
import platform
import shutil
import socket
import subprocess
import sys
from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# FROZEN FINAL STAGE26 IDENTITY
# =============================================================================

REPO_URL = (
    "https://github.com/themubasshir/"
    "ids2018-validation-safe-ablation.git"
)

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

FINAL_STAGE26_ANCHOR = (
    "9e8354ecc9cfa72c28aa037e5d2053de422bf7a2"
)

FINAL_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_g2_final_gpu_publication_closure"
)


EXPECTED_FINAL_FILES = {

    "T26_GPU_WARM_INFERENCE_MEMORY.csv":
        "fd16546ba1b2b4130fc2fdff7095302e38a015770feed1965a4d7df8e4ff8597",

    "T26_CPU1_GPU_MATCHED_COMPARISON.csv":
        "24cbd9a22ad58b85782485a0c08109a7606429b8fc09674056ab7c9cf29edc3b",

    "T26_GPU_BACKEND_RESOURCE_STATUS.csv":
        "3f6922821b4d3f01ff8b84405584acea0d8270a5dba7f4402aaa7ee1fafc9ade",

    "T26_CPU1_GPU_BATCH1_ANCHOR.csv":
        "cc3479a5dadf897fa8beb5eddc2a05ea7e4ed9548e7e7efe0aecc04f09efcfc8",

    "F26_CPU1_GPU_P95_SPEEDUP.png":
        "a5b23cd1513d21798cad7cf1aef3c85e974bbe8086b6d1a868125d2b05b4ced7",

    "F26_CPU1_GPU_P95_SPEEDUP.pdf":
        "a5fa474c6d260521b793e2c380166448e4eb7c055d3b93920601f18b865c44c5",

    "F26_GPU_DELTA_PEAK_MEMORY.png":
        "b5848e0b9bed6c5393feec835c51c120a77bcb8bb7fa0ef1f60ecff10afab24b",

    "F26_GPU_DELTA_PEAK_MEMORY.pdf":
        "55dae00f6eb49ed0c136832a383c9595dbd1790e0c98a986eab901656966ab8b",

    "stage26_g2_final_gpu_publication_index.json":
        "8bb41ad37e3558014ff0d4209dc8d75f323a109ff94462b23fc9a9290462b39a",

    "stage26_g2_final_stage26_closure_receipt.json":
        "e564bc9408b3bb61596e96442e604b1cee03277718015cfd78a0811a47871f65",

    "stage26_g2_final_gpu_publication_closure_manifest.json":
        "1bc78ff5b9d5a4ebae17aa9b3350824da528a29c6e33b44729010cce17ce6bec",
}


# Earlier durable CPU publication package.
CPU_TABLE_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_8d3_cpu_publication_tables"
)

CPU_FIGURE_DIR = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_8d4_cpu_publication_figures"
)


# Critical frozen models.
CRITICAL_MODELS = [

    (
        "XGBoost",
        REPO
        / "results/stage16_classical_benchmark_checkpoint/"
          "stage16_3_tuned_models/XGBOOST_tuned.joblib",
        864264,
        "4f02e06fc6d855adc1e33ce47befda2bedb6c2a5b528338cc88f24e4184b705f",
    ),

    (
        "LightGBM",
        REPO
        / "results/stage16_classical_benchmark_checkpoint/"
          "stage16_3_tuned_models/LIGHTGBM_tuned.joblib",
        2420416,
        "0e13bc2386e30a31aba765501011bd5b274714f77d3a8df481e374a461db55e2",
    ),

    (
        "CatBoost",
        REPO
        / "results/stage16_classical_benchmark_checkpoint/"
          "stage16_3_tuned_models/CATBOOST_tuned.joblib",
        458589,
        "85b2305676280fee6e620d94eb28a58ba9b607aff247b65ad462ab16f89c0cd6",
    ),

    (
        "CNN",
        REPO
        / "results/stage20_1e_training/"
          "stage20_1e2_epoch10_model_state_dict.pt",
        376879,
        "3ebc71e579dc8e0e545981b2d60eea643148fe53e0902f8df8e47556243ad30b",
    ),

    (
        "ViT",
        REPO
        / "results/stage21_architecture/"
          "stage21_2_epoch10_model_state_dict.pt",
        378999,
        "221e9c805fb663acacf2f0f2ca95dba7cb4b2ec4c4de5a3650cf4adeb99b5ef8",
    ),
]


# =============================================================================
# HELPERS
# =============================================================================

def banner(text):
    print()
    print("=" * 122)
    print(text)
    print("=" * 122)


def run(cmd, *, cwd=None, check=True):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(map(str, cmd))}\n\n{p.stdout}"
        )

    return p.stdout.strip()


def git(*args, check=True):
    return run(
        ["git", *args],
        cwd=REPO,
        check=check,
    )


def sha256_file(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for block in iter(
            lambda: f.read(chunk),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def human_bytes(n):
    n = int(n)

    if n >= 1024 ** 3:
        return f"{n / 1024**3:.3f} GiB"

    return f"{n / 1024**2:.3f} MiB"


# =============================================================================
# 1. FRESH CPU RUNTIME
# =============================================================================

banner("STAGE26-R0 :: FRESH CPU RUNTIME")

print("timestamp_utc :", datetime.now(timezone.utc).isoformat())
print("hostname      :", socket.gethostname())
print("python        :", sys.version.replace("\n", " "))
print("platform      :", platform.platform())
print("machine       :", platform.machine())
print("working dir   :", os.getcwd())
print("/kaggle/input :", Path("/kaggle/input").exists())
print("/kaggle/working:", Path("/kaggle/working").exists())


# GPU presence is informational only now.
nvidia = subprocess.run(
    ["nvidia-smi"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    check=False,
)

print()
print(
    "NVIDIA accelerator visible:",
    nvidia.returncode == 0,
)

if nvidia.returncode == 0:
    print(
        "NOTE: Stage26 is already closed; "
        "no GPU measurement is permitted or required."
    )


# =============================================================================
# 2. RESTORE REPOSITORY
# =============================================================================

banner("STAGE26-R0 :: RESTORE FINAL STAGE26 REPOSITORY")


if REPO.exists():

    if not (REPO / ".git").is_dir():

        raise RuntimeError(
            f"{REPO} exists but is not a Git repository."
        )

    status = git(
        "status",
        "--porcelain",
    )

    if status:

        print(status)

        raise RuntimeError(
            "Existing repository is dirty. "
            "Refusing to delete scientific state automatically."
        )

    print("Existing clean repository detected.")

    git(
        "fetch",
        "--prune",
        "origin",
    )


else:

    print("Fresh clone required.")

    run(
        [
            "git",
            "clone",
            REPO_URL,
            str(REPO),
        ],
        cwd=Path("/kaggle/working"),
    )


git(
    "fetch",
    "--prune",
    "origin",
)


# Ensure the frozen final commit exists.
probe = git(
    "cat-file",
    "-e",
    f"{FINAL_STAGE26_ANCHOR}^{{commit}}",
    check=False,
)


if probe is None:
    pass


# Exact checkout.
git(
    "checkout",
    "-B",
    "main",
    FINAL_STAGE26_ANCHOR,
)


head = git(
    "rev-parse",
    "HEAD",
)

origin_main = git(
    "rev-parse",
    "origin/main",
)

origin_url = git(
    "remote",
    "get-url",
    "origin",
)

status = git(
    "status",
    "--porcelain",
)


print("Expected anchor :", FINAL_STAGE26_ANCHOR)
print("Local HEAD      :", head)
print("origin/main     :", origin_main)
print("Origin URL      :", origin_url)
print("Repo clean      :", status == "")


if head != FINAL_STAGE26_ANCHOR:
    raise RuntimeError(
        "Local HEAD is not the final Stage26 anchor."
    )


if origin_main != FINAL_STAGE26_ANCHOR:
    raise RuntimeError(
        "origin/main is no longer the frozen final Stage26 anchor."
    )


if status:
    raise RuntimeError(
        "Repository is not clean after rebuild."
    )


# =============================================================================
# 3. RESTORE REPOSITORY-LOCAL GIT IDENTITY
# =============================================================================

banner("STAGE26-R0 :: RESTORE LOCAL GIT IDENTITY")


author_name = git(
    "show",
    "-s",
    "--format=%an",
    FINAL_STAGE26_ANCHOR,
)

author_email = git(
    "show",
    "-s",
    "--format=%ae",
    FINAL_STAGE26_ANCHOR,
)


git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


print(
    "Git user.name :",
    git(
        "config",
        "--local",
        "--get",
        "user.name",
    ),
)

print(
    "Git user.email:",
    git(
        "config",
        "--local",
        "--get",
        "user.email",
    ),
)


# =============================================================================
# 4. VERIFY FINAL G2 PUBLICATION CLOSURE
# =============================================================================

banner("STAGE26-R0 :: VERIFY FINAL STAGE26 CLOSURE")


if not FINAL_DIR.is_dir():
    raise RuntimeError(
        f"Missing final Stage26 closure directory: {FINAL_DIR}"
    )


for filename, expected_sha in EXPECTED_FINAL_FILES.items():

    path = FINAL_DIR / filename

    if not path.is_file():
        raise FileNotFoundError(
            path
        )

    actual_sha = sha256_file(
        path
    )

    print(
        f"{filename:62s} "
        f"{'PASS' if actual_sha == expected_sha else 'FAIL'}"
    )

    if actual_sha != expected_sha:
        print(
            "  expected:",
            expected_sha,
        )

        print(
            "  actual  :",
            actual_sha,
        )

        raise RuntimeError(
            f"Final Stage26 closure hash mismatch: {filename}"
        )


receipt_path = (
    FINAL_DIR
    / "stage26_g2_final_stage26_closure_receipt.json"
)


with receipt_path.open(
    "r",
    encoding="utf-8",
) as f:
    receipt = json.load(
        f
    )


print()
print(
    "Closure receipt schema:",
    receipt.get(
        "schema"
    ),
)

print(
    "Final Stage26 status :",
    receipt.get(
        "stage26_status",
        receipt.get(
            "status",
            "SEE_RECEIPT",
        ),
    ),
)


# =============================================================================
# 5. VERIFY CPU PUBLICATION PACKAGE
# =============================================================================

banner("STAGE26-R0 :: VERIFY CPU PUBLICATION PACKAGE PRESENCE")


cpu_tables = sorted(
    CPU_TABLE_DIR.glob(
        "T26_*.csv"
    )
)

cpu_png = sorted(
    CPU_FIGURE_DIR.glob(
        "*.png"
    )
)

cpu_pdf = sorted(
    CPU_FIGURE_DIR.glob(
        "*.pdf"
    )
)


print(
    "CPU publication tables :",
    len(cpu_tables),
)

print(
    "CPU PNG figures        :",
    len(cpu_png),
)

print(
    "CPU PDF figures        :",
    len(cpu_pdf),
)


if len(cpu_tables) != 8:
    raise RuntimeError(
        "Expected 8 frozen CPU publication tables."
    )


if len(cpu_png) != 8:
    raise RuntimeError(
        "Expected 8 frozen CPU PNG figures."
    )


if len(cpu_pdf) != 8:
    raise RuntimeError(
        "Expected 8 frozen CPU PDF figures."
    )


# =============================================================================
# 6. VERIFY CRITICAL MODEL ARTIFACTS
# =============================================================================

banner("STAGE26-R0 :: VERIFY CRITICAL FROZEN MODELS")


for (
    name,
    path,
    expected_size,
    expected_sha,
) in CRITICAL_MODELS:

    if not path.is_file():
        raise FileNotFoundError(
            path
        )

    size = path.stat().st_size

    actual_sha = sha256_file(
        path
    )


    ok = (
        size == expected_size
        and
        actual_sha == expected_sha
    )


    print(
        f"{name:12s}: "
        f"{'PASS' if ok else 'FAIL'}  "
        f"{human_bytes(size)}"
    )


    if not ok:

        print(
            " expected size:",
            expected_size,
        )

        print(
            " actual size  :",
            size,
        )

        print(
            " expected SHA :",
            expected_sha,
        )

        print(
            " actual SHA   :",
            actual_sha,
        )

        raise RuntimeError(
            f"Frozen model verification failed: {name}"
        )


# =============================================================================
# 7. CHECK GITHUB SECRET FOR FUTURE WRITES
# =============================================================================

banner("STAGE26-R0 :: GITHUB WRITE CAPABILITY")


github_secret_label = None


try:

    from kaggle_secrets import UserSecretsClient

    usc = UserSecretsClient()


    for label in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = usc.get_secret(
                label
            )

            if value:

                github_secret_label = label
                break

        except Exception:
            pass


except Exception as exc:

    print(
        "Kaggle Secrets warning:",
        repr(exc),
    )


print(
    "Usable GitHub secret:",
    github_secret_label or "NOT_FOUND",
)


# =============================================================================
# 8. CORPUS POLICY
# =============================================================================

banner("STAGE26-R0 :: CORPUS / DATA POLICY")


print(
    "Compact corpus restored       : NO"
)

print(
    "Compact corpus required now   : NO"
)

print(
    "Raw Monday PCAP restoration   : NO"
)

print(
    "Corpus reconstruction         : PROHIBITED"
)

print()
print(
    "Reason:"
)

print(
    "  Stage26 measurement science is already COMPLETE and FROZEN."
)

print(
    "  The next phase uses durable publication tables, figures, "
    "receipts, and manifests."
)

print(
    "  Restore the compact corpus only if a future preregistered "
    "scientific stage explicitly requires it."
)


# =============================================================================
# 9. FINAL REBUILD STATE
# =============================================================================

banner("STAGE26-R0 COMPLETE :: POST-STAGE26 WORKSPACE REBUILT")


final_status = git(
    "status",
    "--porcelain",
)


print(
    "FINAL_STAGE26_ANCHOR :",
    FINAL_STAGE26_ANCHOR,
)

print(
    "HEAD == origin/main  :",
    (
        head
        ==
        origin_main
        ==
        FINAL_STAGE26_ANCHOR
    ),
)

print(
    "Repo clean           :",
    final_status == "",
)

print()
print(
    "CPU publication tables:",
    len(cpu_tables),
    "/ 8"
)

print(
    "CPU publication PNGs  :",
    len(cpu_png),
    "/ 8"
)

print(
    "CPU publication PDFs  :",
    len(cpu_pdf),
    "/ 8"
)

print(
    "Final G2 files        :",
    len(EXPECTED_FINAL_FILES),
    "/ 11"
)

print(
    "Critical models       :",
    len(CRITICAL_MODELS),
    "/",
    len(CRITICAL_MODELS),
)

print()
print(
    "STAGE26 SCIENCE       : FROZEN / DO NOT RERUN"
)

print(
    "WORKSPACE STATUS      : READY"
)

print(
    "NEXT PHASE            : PAPER-FACING ANALYSIS / MANUSCRIPT"
)


if final_status:
    raise RuntimeError(
        "Repository not clean after rebuild."
    )


STAGE26-R0 :: FRESH CPU RUNTIME
timestamp_utc : 2026-08-20T10:57:17.570645+00:00
hostname      : 3ac5dadb3f82
python        : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
platform      : Linux-6.12.90+-x86_64-with-glibc2.35
machine       : x86_64
working dir   : /kaggle/working
/kaggle/input : True
/kaggle/working: True

NVIDIA accelerator visible: False

STAGE26-R0 :: RESTORE FINAL STAGE26 REPOSITORY
Fresh clone required.
Expected anchor : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
Local HEAD      : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
origin/main     : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
Origin URL      : https://github.com/themubasshir/ids2018-validation-safe-ablation.git
Repo clean      : True

STAGE26-R0 :: RESTORE LOCAL GIT IDENTITY
Git user.name : themubasshir
Git user.email: themubasshir@users.noreply.github.com

STAGE26-R0 :: VERIFY FINAL STAGE26 CLOSURE
T26_GPU_WARM_INFERENCE_MEMORY.csv                              PASS
T26_CPU1_GPU_MATCHED_COMPARISON.csv       

In [2]:
# =============================================================================
# STAGE26-R0R1 :: CPU-RUNTIME NVIDIA-SMI ABSENCE RECOVERY
#
# Purpose:
#   Make the already-written R0 informational GPU check safe on a true CPU-only
#   Kaggle runtime where the nvidia-smi executable does not exist.
#
# No repository access.
# No science.
# No measurement.
# No GPU.
# =============================================================================

from pathlib import Path
import os
import shutil

print("=" * 110)
print("STAGE26-R0R1 :: CPU-ONLY NVIDIA TOOLING RECOVERY")
print("=" * 110)

real_nvidia_smi = shutil.which("nvidia-smi")

print("Real nvidia-smi :", real_nvidia_smi or "NOT INSTALLED")
print("GPU expected    : NO")
print("Scientific state: UNCHANGED")

if real_nvidia_smi is None:
    # The original R0 code only checks the return code from nvidia-smi.
    # Provide a harmless local command that returns non-zero so R0 records:
    # NVIDIA accelerator visible: False
    #
    # This does NOT emulate a GPU and cannot be mistaken for one.
    stub_dir = Path("/kaggle/working/.stage26_cpu_only_bin")
    stub_dir.mkdir(parents=True, exist_ok=True)

    stub = stub_dir / "nvidia-smi"

    stub.write_text(
        "#!/bin/sh\n"
        "echo 'nvidia-smi unavailable: CPU-only Stage26 post-closure runtime' >&2\n"
        "exit 127\n",
        encoding="utf-8",
    )

    stub.chmod(0o755)

    current_path = os.environ.get("PATH", "")

    if str(stub_dir) not in current_path.split(":"):
        os.environ["PATH"] = f"{stub_dir}:{current_path}"

    print("CPU-only compatibility stub:", stub)
    print("PATH patched for this kernel : YES")

else:
    print("Compatibility stub required : NO")

resolved = shutil.which("nvidia-smi")

print("Resolved nvidia-smi command :", resolved)

if resolved is None:
    raise RuntimeError("Recovery failed to provide a safe nvidia-smi command.")

print()
print("RECOVERY_STATUS : PASS")
print("REPO_MODIFIED   : NO")
print("SCIENCE_CHANGED : NO")
print("GPU_USED        : NO")
print()
print("NEXT: rerun the SAME Stage26-R0 rebuild cell from the beginning.")

STAGE26-R0R1 :: CPU-ONLY NVIDIA TOOLING RECOVERY
Real nvidia-smi : NOT INSTALLED
GPU expected    : NO
Scientific state: UNCHANGED
CPU-only compatibility stub: /kaggle/working/.stage26_cpu_only_bin/nvidia-smi
PATH patched for this kernel : YES
Resolved nvidia-smi command : /kaggle/working/.stage26_cpu_only_bin/nvidia-smi

RECOVERY_STATUS : PASS
REPO_MODIFIED   : NO
SCIENCE_CHANGED : NO
GPU_USED        : NO

NEXT: rerun the SAME Stage26-R0 rebuild cell from the beginning.


In [4]:
# =============================================================================
# STAGE26-P0 :: PAPER-FACING DEPLOYMENT SYNTHESIS
#
# READ-ONLY SCIENTIFIC SYNTHESIS
#
# - NO inference
# - NO timing
# - NO model loading
# - NO corpus access
# - NO Git modification
# - NO new scientific measurement
#
# Produces only a temporary manuscript-analysis preview under /kaggle/working.
# =============================================================================

from __future__ import annotations

import json
import hashlib
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd


REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

FINAL_STAGE26_ANCHOR = (
    "9e8354ecc9cfa72c28aa037e5d2053de422bf7a2"
)

G2 = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_g2_final_gpu_publication_closure"
)

CPU_TABLES = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_8d3_cpu_publication_tables"
)

OUT = Path(
    "/kaggle/working/stage26_paper_synthesis_preview"
)


PRIMARY_TARGETS = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
]

REFERENCE_TARGETS = [
    "ENS_LGBM_XGB_EQUAL",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE",
]

DISPLAY = {
    "STAGE16_XGBOOST_TUNED": "XGBoost",
    "STAGE16_LIGHTGBM_TUNED": "LightGBM",
    "STAGE16_CATBOOST_TUNED": "CatBoost",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING": "FT Transformer (5-checkpoint ensemble)",
    "STAGE20_MASKED_CNN_V1": "Masked CNN",
    "STAGE21_MASKED_VIT_V1": "Masked ViT",
    "ENS_LGBM_XGB_EQUAL": "LightGBM+XGBoost ensemble",
    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE": "FT Transformer (single checkpoint reference)",
}


def banner(text):
    print()
    print("=" * 122)
    print(text)
    print("=" * 122)


def git(*args):
    p = subprocess.run(
        ["git", *args],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(p.stdout)

    return p.stdout.strip()


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for block in iter(
            lambda: f.read(16 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def fmt(x, digits=3):
    if pd.isna(x):
        return "N/A"

    return f"{float(x):.{digits}f}"


# =============================================================================
# 1. FROZEN STATE GATE
# =============================================================================

banner("STAGE26-P0 :: FROZEN REPOSITORY GATE")

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)

print("Expected anchor :", FINAL_STAGE26_ANCHOR)
print("Local HEAD      :", head)
print("origin/main     :", origin)
print("Repo clean      :", status == "")

if head != FINAL_STAGE26_ANCHOR:
    raise RuntimeError(
        "Unexpected local HEAD."
    )

if origin != FINAL_STAGE26_ANCHOR:
    raise RuntimeError(
        "Unexpected origin/main."
    )

if status:
    raise RuntimeError(
        "Repository is dirty before manuscript synthesis."
    )


# =============================================================================
# 2. LOAD FROZEN PUBLICATION SOURCES
# =============================================================================

banner("STAGE26-P0 :: LOAD FINAL PUBLICATION SOURCES")

paths = {
    "matched":
        G2 / "T26_CPU1_GPU_MATCHED_COMPARISON.csv",

    "b1":
        G2 / "T26_CPU1_GPU_BATCH1_ANCHOR.csv",

    "gpu_status":
        G2 / "T26_GPU_BACKEND_RESOURCE_STATUS.csv",

    "gpu_warm":
        G2 / "T26_GPU_WARM_INFERENCE_MEMORY.csv",

    "cpu_pareto":
        CPU_TABLES / "T26_PARETO.csv",
}


for key, path in paths.items():
    print(
        f"{key:12s}:",
        path.relative_to(REPO)
    )

    if not path.is_file():
        raise FileNotFoundError(
            path
        )


matched = pd.read_csv(
    paths["matched"]
)

b1 = pd.read_csv(
    paths["b1"]
)

gpu_status = pd.read_csv(
    paths["gpu_status"]
)

gpu_warm = pd.read_csv(
    paths["gpu_warm"]
)

cpu_pareto = pd.read_csv(
    paths["cpu_pareto"]
)


print()
print("Matched CPU1/GPU rows :", len(matched))
print("B1 anchor rows        :", len(b1))
print("GPU warm rows         :", len(gpu_warm))
print("GPU status rows       :", len(gpu_status))
print("CPU Pareto rows       :", len(cpu_pareto))


if len(matched) != 26:
    raise RuntimeError(
        f"Expected 26 matched CPU1/GPU PASS rows, got {len(matched)}"
    )

if len(b1) != 6:
    raise RuntimeError(
        f"Expected 6 matched B1 anchor rows, got {len(b1)}"
    )

if len(gpu_warm) != 40:
    raise RuntimeError(
        f"Expected 40 GPU geometry rows, got {len(gpu_warm)}"
    )

if len(gpu_status) != 8:
    raise RuntimeError(
        f"Expected 8 GPU target status rows, got {len(gpu_status)}"
    )

if len(cpu_pareto) != 6:
    raise RuntimeError(
        f"Expected 6 CPU Pareto rows, got {len(cpu_pareto)}"
    )


# =============================================================================
# 3. BATCH-1 DEPLOYMENT ANCHOR
# =============================================================================

banner("STAGE26-P0 :: BATCH-1 CPU1 ↔ GPU DEPLOYMENT ANCHOR")

b1 = b1.copy()

b1["model"] = (
    b1["target_id"]
    .map(DISPLAY)
    .fillna(b1["target_id"])
)

b1["winner_p95"] = np.where(
    b1["p95_latency_speedup_cpu_over_gpu"] > 1.0,
    "GPU",
    "CPU1",
)


cols = [
    "model",
    "cpu_p95_latency_ms",
    "gpu_p95_latency_ms",
    "p95_latency_speedup_cpu_over_gpu",
    "cpu_median_throughput_samples_per_s",
    "gpu_median_throughput_samples_per_s",
    "winner_p95",
]


print(
    b1[cols]
    .sort_values(
        "p95_latency_speedup_cpu_over_gpu",
        ascending=False,
    )
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# =============================================================================
# 4. BATCHWISE CPU/GPU CROSSOVER ANALYSIS
# =============================================================================

banner("STAGE26-P0 :: BATCHWISE CPU/GPU CROSSOVER")

crossover_records = []


for target, group in matched.groupby(
    "target_id",
    sort=False,
):

    group = group.sort_values(
        "batch_size"
    )

    model = DISPLAY.get(
        target,
        target,
    )

    ratios = []

    for _, row in group.iterrows():

        ratios.append(
            {
                "batch_size":
                    int(row["batch_size"]),

                "p95_speedup":
                    float(
                        row[
                            "p95_latency_speedup_cpu_over_gpu"
                        ]
                    ),
            }
        )

    gpu_adv = [
        x
        for x in ratios
        if x["p95_speedup"] > 1.0
    ]

    cpu_adv = [
        x
        for x in ratios
        if x["p95_speedup"] < 1.0
    ]

    if gpu_adv:
        first_gpu_batch = min(
            x["batch_size"]
            for x in gpu_adv
        )
    else:
        first_gpu_batch = None

    if gpu_adv and cpu_adv:
        pattern = "BATCH_DEPENDENT_CROSSOVER"

    elif gpu_adv and not cpu_adv:
        pattern = "GPU_ADVANTAGE_ALL_MATCHED_BATCHES"

    elif cpu_adv and not gpu_adv:
        pattern = "CPU_ADVANTAGE_ALL_MATCHED_BATCHES"

    else:
        pattern = "PARITY_ONLY"

    crossover_records.append(
        {
            "target_id":
                target,

            "model":
                model,

            "matched_batches":
                [
                    x["batch_size"]
                    for x in ratios
                ],

            "p95_speedups_cpu_over_gpu":
                [
                    x["p95_speedup"]
                    for x in ratios
                ],

            "first_matched_batch_with_gpu_advantage":
                first_gpu_batch,

            "pattern":
                pattern,
        }
    )


for rec in crossover_records:

    ratio_text = ", ".join(
        f"B{x['batch_size']}={x['p95_speedup']:.3f}x"
        for x in [
            {
                "batch_size": b,
                "p95_speedup": s,
            }
            for b, s in zip(
                rec["matched_batches"],
                rec["p95_speedups_cpu_over_gpu"],
            )
        ]
    )

    print()
    print(rec["model"])
    print("  pattern :", rec["pattern"])
    print("  ratios  :", ratio_text)
    print(
        "  first GPU-advantage batch:",
        rec[
            "first_matched_batch_with_gpu_advantage"
        ],
    )


# =============================================================================
# 5. GPU MEMORY / RESOURCE LIMIT ANALYSIS
# =============================================================================

banner("STAGE26-P0 :: GPU MEMORY / RESOURCE LIMIT STORY")

gpu_pass = gpu_warm[
    gpu_warm["status"] == "PASS"
].copy()


memory_records = []


for target, group in gpu_pass.groupby(
    "target_id",
    sort=False,
):

    valid = group.dropna(
        subset=["delta_peak_gpu_memory_mib"]
    )

    if valid.empty:
        continue

    idx = valid[
        "delta_peak_gpu_memory_mib"
    ].idxmax()

    row = valid.loc[idx]

    memory_records.append(
        {
            "target_id":
                target,

            "model":
                DISPLAY.get(
                    target,
                    target,
                ),

            "max_observed_pass_batch":
                int(
                    group["batch_size"].max()
                ),

            "max_delta_peak_gpu_memory_mib":
                float(
                    row[
                        "delta_peak_gpu_memory_mib"
                    ]
                ),

            "batch_at_max_delta_peak":
                int(
                    row["batch_size"]
                ),
        }
    )


memory_df = pd.DataFrame(
    memory_records
)


print(
    memory_df.sort_values(
        "max_delta_peak_gpu_memory_mib",
        ascending=False,
    ).to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}",
    )
)


non_pass = gpu_warm[
    gpu_warm["status"] != "PASS"
][
    [
        "target_id",
        "batch_size",
        "status",
        "error_or_backend_detail",
    ]
].copy()


non_pass["model"] = (
    non_pass["target_id"]
    .map(DISPLAY)
    .fillna(non_pass["target_id"])
)


print()
print("NON-PASS GPU CONDITIONS")
print(
    non_pass[
        [
            "model",
            "batch_size",
            "status",
        ]
    ].to_string(
        index=False
    )
)


# =============================================================================
# 6. CPU PARETO CONTEXT
# =============================================================================

banner("STAGE26-P0 :: FROZEN CPU PARETO CONTEXT")

pareto = cpu_pareto.copy()

pareto["model"] = (
    pareto["target_id"]
    .map(DISPLAY)
    .fillna(pareto["target_id"])
)


print(
    pareto[
        [
            "group",
            "model",
            "pr_auc",
            "cpu1_batch1_p95_latency_ms",
            "pareto_status",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


# =============================================================================
# 7. DERIVE PAPER-FACING CLAIMS
# =============================================================================

banner("STAGE26-P0 :: PAPER-FACING CLAIM REGISTRY")

b1_lookup = {
    row["target_id"]: row
    for _, row in b1.iterrows()
}


claims = []


def add_claim(
    claim_id,
    category,
    wording,
    evidence,
    strength,
):
    claims.append(
        {
            "claim_id": claim_id,
            "category": category,
            "wording": wording,
            "evidence": evidence,
            "strength": strength,
        }
    )


# CNN
cnn = b1_lookup[
    "STAGE20_MASKED_CNN_V1"
]

add_claim(
    "C26_DEPLOY_01",
    "CPU_GPU_LATENCY",
    (
        "For the frozen masked CNN at batch size 1, "
        "single-T4 GPU inference reduced p95 component-level "
        "latency relative to CPU1 by approximately "
        f"{cnn['p95_latency_speedup_cpu_over_gpu']:.2f}×."
    ),
    (
        f"CPU1 p95={cnn['cpu_p95_latency_ms']:.6f} ms; "
        f"GPU p95={cnn['gpu_p95_latency_ms']:.6f} ms."
    ),
    "HIGH",
)


# FT ensemble
ft = b1_lookup[
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING"
]

add_claim(
    "C26_DEPLOY_02",
    "CPU_GPU_LATENCY",
    (
        "The five-checkpoint FT Transformer ensemble also benefited "
        "from GPU execution at batch size 1, with approximately "
        f"{ft['p95_latency_speedup_cpu_over_gpu']:.2f}× lower "
        "p95 component-level inference latency relative to CPU1."
    ),
    (
        f"CPU1 p95={ft['cpu_p95_latency_ms']:.6f} ms; "
        f"GPU p95={ft['gpu_p95_latency_ms']:.6f} ms."
    ),
    "HIGH",
)


# XGB
xgb = b1_lookup[
    "STAGE16_XGBOOST_TUNED"
]

add_claim(
    "C26_DEPLOY_03",
    "SMALL_BATCH_CPU_ADVANTAGE",
    (
        "GPU execution was not universally advantageous: "
        "at batch size 1 the frozen XGBoost model had lower "
        "p95 latency on CPU1 than on the T4."
    ),
    (
        f"CPU1 p95={xgb['cpu_p95_latency_ms']:.6f} ms; "
        f"GPU p95={xgb['gpu_p95_latency_ms']:.6f} ms; "
        f"CPU/GPU p95 ratio={xgb['p95_latency_speedup_cpu_over_gpu']:.3f}."
    ),
    "HIGH",
)


# CatBoost
cat = b1_lookup[
    "STAGE16_CATBOOST_TUNED"
]

add_claim(
    "C26_DEPLOY_04",
    "SMALL_BATCH_CPU_ADVANTAGE",
    (
        "CatBoost showed an even stronger small-batch CPU advantage: "
        "batch-1 p95 inference latency was substantially lower on CPU1 "
        "than on the T4."
    ),
    (
        f"CPU1 p95={cat['cpu_p95_latency_ms']:.6f} ms; "
        f"GPU p95={cat['gpu_p95_latency_ms']:.6f} ms; "
        f"CPU/GPU p95 ratio={cat['p95_latency_speedup_cpu_over_gpu']:.3f}."
    ),
    "HIGH",
)


# ViT
vit = b1_lookup[
    "STAGE21_MASKED_VIT_V1"
]

add_claim(
    "C26_DEPLOY_05",
    "NEAR_PARITY",
    (
        "The masked ViT was near parity at batch size 1, "
        "with only a small point-estimate p95 advantage for the GPU."
    ),
    (
        f"CPU1 p95={vit['cpu_p95_latency_ms']:.6f} ms; "
        f"GPU p95={vit['gpu_p95_latency_ms']:.6f} ms; "
        f"CPU/GPU p95 ratio={vit['p95_latency_speedup_cpu_over_gpu']:.3f}."
    ),
    "HIGH",
)


# Backend availability
add_claim(
    "C26_DEPLOY_06",
    "BACKEND_AVAILABILITY",
    (
        "The frozen LightGBM deployment path did not expose a "
        "contract-approved native GPU inference backend; consequently "
        "the LightGBM+XGBoost operational ensemble was also unavailable "
        "for complete GPU profiling."
    ),
    "10 GPU conditions recorded as BACKEND_UNAVAILABLE without substitution.",
    "HIGH",
)


# CNN OOM
add_claim(
    "C26_DEPLOY_07",
    "RESOURCE_LIMIT",
    (
        "The masked CNN reached a resource limit at batch size 8192 "
        "on the 14.56-GiB Tesla T4 and was preserved as a CUDA OOM "
        "rather than assigned an imputed deployment cost."
    ),
    "GPU CNN B8192 status=RESOURCE_LIMIT_OOM.",
    "HIGH",
)


for claim in claims:

    print()
    print(
        claim["claim_id"],
        f"[{claim['strength']}]"
    )

    print(
        " ",
        claim["wording"]
    )

    print(
        "  Evidence:",
        claim["evidence"]
    )


# =============================================================================
# 8. PROHIBITED / REQUIRED QUALIFIERS
# =============================================================================

banner("STAGE26-P0 :: CLAIM-SAFETY BOUNDARIES")

boundaries = [
    "Use 'component-level inference' rather than complete pipeline latency.",
    "Do not claim complete end-to-end throughput; complete E2E measurement is unavailable.",
    "Do not compare Pareto frontiers across GROUP_A_DUPSAFE70 and GROUP_B_PACKET_IMAGE.",
    "Do not report CPU/GPU ratio confidence intervals; they were not computed.",
    "Do not call a batch size 'optimal' or 'best' based on post-hoc profiling.",
    "Do not impute latency or memory for OOM, timeout, or BACKEND_UNAVAILABLE conditions.",
    "Do not describe LightGBM GPU inference as slow; classify it as BACKEND_UNAVAILABLE.",
    "Do not treat FT_BALANCED_SINGLE_RESOURCE_REFERENCE as a predictive Pareto candidate.",
    "Do not replace historical Stage26-4C3 representation results with the sensitivity implementation.",
    "Use Stage26-6F1 corrected warm CPU uncertainty, not historical Stage26-2 confidence intervals.",
]


for i, text in enumerate(
    boundaries,
    start=1,
):
    print(
        f"{i:02d}. {text}"
    )


# =============================================================================
# 9. RECOMMENDED IEEE PRESENTATION
# =============================================================================

banner("STAGE26-P0 :: RECOMMENDED IEEE MAIN-PAPER CONTENT")

main_paper = [
    {
        "priority": 1,
        "asset": "F26_CPU1_GPU_P95_SPEEDUP",
        "role": (
            "Main deployment figure: directly shows that acceleration "
            "is architecture- and batch-dependent rather than universal."
        ),
    },
    {
        "priority": 2,
        "asset": "T26_CPU1_GPU_BATCH1_ANCHOR",
        "role": (
            "Compact latency/throughput anchor for online or low-batch IDS deployment."
        ),
    },
    {
        "priority": 3,
        "asset": "F26_PARETO",
        "role": (
            "CPU-side accuracy–latency trade-off, retaining separate Group A/B panels."
        ),
    },
    {
        "priority": 4,
        "asset": "F26_GPU_DELTA_PEAK_MEMORY",
        "role": (
            "Deployment feasibility/resource-scaling evidence, including the CNN memory ceiling."
        ),
    },
]


supplement = [
    "T26_GPU_WARM_INFERENCE_MEMORY",
    "T26_CPU1_GPU_MATCHED_COMPARISON",
    "T26_GPU_BACKEND_RESOURCE_STATUS",
    "T26_CPU_WARM_INFERENCE",
    "T26_CPU_MEMORY_PACKAGE",
    "T26_CPU_CAPACITY_SCALING",
    "T26_REPRESENTATION_SENSITIVITY",
    "F26_COMPONENT_BOUNDARY",
]


print("MAIN PAPER")
for item in main_paper:
    print(
        f"  {item['priority']}. "
        f"{item['asset']}: "
        f"{item['role']}"
    )


print()
print("SUPPLEMENT / APPENDIX")

for item in supplement:
    print(
        "  -",
        item,
    )


# =============================================================================
# 10. WRITE TEMPORARY SYNTHESIS PREVIEW
# =============================================================================

banner("STAGE26-P0 :: WRITE TEMPORARY SYNTHESIS PREVIEW")

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


claim_df = pd.DataFrame(
    claims
)

claim_csv = (
    OUT
    / "stage26_paper_claim_registry.csv"
)

claim_df.to_csv(
    claim_csv,
    index=False,
)


crossover_df = pd.DataFrame(
    crossover_records
)

# Serialize lists cleanly for CSV.
for col in [
    "matched_batches",
    "p95_speedups_cpu_over_gpu",
]:
    crossover_df[col] = crossover_df[col].apply(
        json.dumps
    )


crossover_csv = (
    OUT
    / "stage26_cpu_gpu_crossover_summary.csv"
)

crossover_df.to_csv(
    crossover_csv,
    index=False,
)


summary = {
    "schema":
        "stage26_postclosure_paper_synthesis_preview_v1",

    "final_stage26_anchor":
        FINAL_STAGE26_ANCHOR,

    "source_policy":
        "FROZEN_STAGE26_PUBLICATION_ARTIFACTS_ONLY",

    "new_measurement_performed":
        False,

    "repository_modified":
        False,

    "matched_cpu1_gpu_pass_rows":
        int(
            len(matched)
        ),

    "batch1_anchor_rows":
        int(
            len(b1)
        ),

    "claims":
        claims,

    "crossover":
        crossover_records,

    "claim_boundaries":
        boundaries,

    "recommended_main_paper":
        main_paper,

    "recommended_supplement":
        supplement,
}


summary_json = (
    OUT
    / "stage26_paper_synthesis_preview.json"
)

summary_json.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)


for path in [
    claim_csv,
    crossover_csv,
    summary_json,
]:

    print()
    print(path.name)
    print(
        "  bytes :",
        path.stat().st_size,
    )
    print(
        "  SHA256:",
        sha256_file(path),
    )


# =============================================================================
# 11. FINAL STATE
# =============================================================================

banner("STAGE26-P0 COMPLETE :: PAPER SYNTHESIS READY")

final_repo_status = git(
    "status",
    "--porcelain",
)


print(
    "Final Stage26 anchor        :",
    FINAL_STAGE26_ANCHOR,
)

print(
    "Frozen publication sources : VERIFIED"
)

print(
    "New timing/inference        : NONE"
)

print(
    "Model loading               : NONE"
)

print(
    "Corpus access               : NONE"
)

print(
    "Repository modified         :",
    bool(final_repo_status),
)

print()
print(
    "Claim registry rows         :",
    len(claims),
)

print(
    "CPU/GPU crossover models    :",
    len(crossover_records),
)

print()
print(
    "TEMP OUTPUT                 :",
    OUT,
)

print()
print(
    "NEXT:"
)

print(
    "  Paste the COMPLETE output."
)

print(
    "  We will turn the verified numerical story into the actual "
    "IEEE Results + Deployment Efficiency + Discussion structure."
)


if final_repo_status:
    raise RuntimeError(
        "Repository unexpectedly changed during read-only synthesis."
    )


STAGE26-P0 :: FROZEN REPOSITORY GATE
Expected anchor : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
Local HEAD      : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
origin/main     : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
Repo clean      : True

STAGE26-P0 :: LOAD FINAL PUBLICATION SOURCES
matched     : results/stage26_deployment_profiling/stage26_g2_final_gpu_publication_closure/T26_CPU1_GPU_MATCHED_COMPARISON.csv
b1          : results/stage26_deployment_profiling/stage26_g2_final_gpu_publication_closure/T26_CPU1_GPU_BATCH1_ANCHOR.csv
gpu_status  : results/stage26_deployment_profiling/stage26_g2_final_gpu_publication_closure/T26_GPU_BACKEND_RESOURCE_STATUS.csv
gpu_warm    : results/stage26_deployment_profiling/stage26_g2_final_gpu_publication_closure/T26_GPU_WARM_INFERENCE_MEMORY.csv
cpu_pareto  : results/stage26_deployment_profiling/stage26_8d3_cpu_publication_tables/T26_PARETO.csv

Matched CPU1/GPU rows : 26
B1 anchor rows        : 6
GPU warm rows         : 40
GPU status rows       : 8
CPU

In [5]:
# =============================================================================
# STAGE26-P1 :: IEEE MANUSCRIPT RESULTS / DEPLOYMENT / DISCUSSION DRAFT
#
# READ-ONLY WITH RESPECT TO THE REPOSITORY
#
# Inputs:
#   - final Stage26 publication tables
#   - Stage26-P0 claim registry / crossover synthesis
#
# Outputs:
#   /kaggle/working/stage26_manuscript_draft/
#       stage26_results_deployment_discussion_draft.md
#       stage26_claim_traceability.csv
#       stage26_manuscript_asset_plan.csv
#
# NO inference
# NO timing
# NO model loading
# NO corpus access
# NO Git commit
# =============================================================================

from __future__ import annotations

import hashlib
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd


# =============================================================================
# FROZEN STATE
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

ANCHOR = (
    "9e8354ecc9cfa72c28aa037e5d2053de422bf7a2"
)

G2 = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_g2_final_gpu_publication_closure"
)

CPU = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_8d3_cpu_publication_tables"
)

P0 = Path(
    "/kaggle/working/stage26_paper_synthesis_preview"
)

OUT = Path(
    "/kaggle/working/stage26_manuscript_draft"
)


DISPLAY = {
    "STAGE16_XGBOOST_TUNED":
        "XGBoost",

    "STAGE16_LIGHTGBM_TUNED":
        "LightGBM",

    "STAGE16_CATBOOST_TUNED":
        "CatBoost",

    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING":
        "FT Transformer ensemble",

    "STAGE20_MASKED_CNN_V1":
        "masked CNN",

    "STAGE21_MASKED_VIT_V1":
        "masked ViT",

    "ENS_LGBM_XGB_EQUAL":
        "LightGBM+XGBoost ensemble",

    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE":
        "single-checkpoint FT reference",
}


PRIMARY = [
    "STAGE16_XGBOOST_TUNED",
    "STAGE16_LIGHTGBM_TUNED",
    "STAGE16_CATBOOST_TUNED",
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    "STAGE20_MASKED_CNN_V1",
    "STAGE21_MASKED_VIT_V1",
]


# =============================================================================
# HELPERS
# =============================================================================

def banner(text):
    print()
    print("=" * 122)
    print(text)
    print("=" * 122)


def git(*args):
    p = subprocess.run(
        ["git", *args],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(p.stdout)

    return p.stdout.strip()


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for block in iter(
            lambda: f.read(16 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def f3(x):
    return f"{float(x):.3f}"


def f2(x):
    return f"{float(x):.2f}"


def f6(x):
    return f"{float(x):.6f}"


def inv_ratio(x):
    x = float(x)

    if x <= 0:
        return np.nan

    return 1.0 / x


# =============================================================================
# 1. FROZEN REPOSITORY GATE
# =============================================================================

banner("STAGE26-P1 :: FROZEN REPOSITORY GATE")

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)

print("Expected anchor :", ANCHOR)
print("Local HEAD      :", head)
print("origin/main     :", origin)
print("Repo clean      :", status == "")

if head != ANCHOR:
    raise RuntimeError(
        "Unexpected local HEAD."
    )

if origin != ANCHOR:
    raise RuntimeError(
        "Unexpected origin/main."
    )

if status:
    raise RuntimeError(
        "Repository is dirty."
    )


# =============================================================================
# 2. LOAD FROZEN PUBLICATION SOURCES
# =============================================================================

banner("STAGE26-P1 :: LOAD PUBLICATION SOURCES")

matched = pd.read_csv(
    G2
    / "T26_CPU1_GPU_MATCHED_COMPARISON.csv"
)

b1 = pd.read_csv(
    G2
    / "T26_CPU1_GPU_BATCH1_ANCHOR.csv"
)

gpu = pd.read_csv(
    G2
    / "T26_GPU_WARM_INFERENCE_MEMORY.csv"
)

gpu_status = pd.read_csv(
    G2
    / "T26_GPU_BACKEND_RESOURCE_STATUS.csv"
)

pareto = pd.read_csv(
    CPU
    / "T26_PARETO.csv"
)

cpu_warm = pd.read_csv(
    CPU
    / "T26_CPU_WARM_INFERENCE.csv"
)


claim_registry = pd.read_csv(
    P0
    / "stage26_paper_claim_registry.csv"
)


print("Matched CPU/GPU rows :", len(matched))
print("B1 rows              :", len(b1))
print("GPU rows             :", len(gpu))
print("GPU status rows      :", len(gpu_status))
print("CPU Pareto rows      :", len(pareto))
print("CPU warm rows        :", len(cpu_warm))
print("P0 claim rows        :", len(claim_registry))


if len(matched) != 26:
    raise RuntimeError(
        "Matched comparison geometry changed."
    )

if len(b1) != 6:
    raise RuntimeError(
        "B1 comparison geometry changed."
    )

if len(gpu) != 40:
    raise RuntimeError(
        "GPU geometry changed."
    )

if len(pareto) != 6:
    raise RuntimeError(
        "Pareto geometry changed."
    )


# =============================================================================
# 3. RESOLVE KEY VALUES
# =============================================================================

banner("STAGE26-P1 :: RESOLVE MANUSCRIPT VALUES")

b1_map = {
    r["target_id"]: r
    for _, r in b1.iterrows()
}


# -------------------------------
# Batch-1 anchors
# -------------------------------

xgb = b1_map[
    "STAGE16_XGBOOST_TUNED"
]

cat = b1_map[
    "STAGE16_CATBOOST_TUNED"
]

ft = b1_map[
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING"
]

cnn = b1_map[
    "STAGE20_MASKED_CNN_V1"
]

vit = b1_map[
    "STAGE21_MASKED_VIT_V1"
]


xgb_cpu_adv = inv_ratio(
    xgb[
        "p95_latency_speedup_cpu_over_gpu"
    ]
)

cat_cpu_adv = inv_ratio(
    cat[
        "p95_latency_speedup_cpu_over_gpu"
    ]
)


# -------------------------------
# Crossovers
# -------------------------------

def row_for(target, batch):
    z = matched[
        (matched["target_id"] == target)
        &
        (matched["batch_size"] == batch)
    ]

    if len(z) != 1:
        raise RuntimeError(
            f"Expected one row for {target} B{batch}, got {len(z)}"
        )

    return z.iloc[0]


xgb_b256 = row_for(
    "STAGE16_XGBOOST_TUNED",
    256,
)

xgb_b1024 = row_for(
    "STAGE16_XGBOOST_TUNED",
    1024,
)

xgb_b8192 = row_for(
    "STAGE16_XGBOOST_TUNED",
    8192,
)

cat_b1024 = row_for(
    "STAGE16_CATBOOST_TUNED",
    1024,
)

cat_b8192 = row_for(
    "STAGE16_CATBOOST_TUNED",
    8192,
)

cnn_b64 = row_for(
    "STAGE20_MASKED_CNN_V1",
    64,
)

cnn_b256 = row_for(
    "STAGE20_MASKED_CNN_V1",
    256,
)

vit_b64 = row_for(
    "STAGE21_MASKED_VIT_V1",
    64,
)

vit_b256 = row_for(
    "STAGE21_MASKED_VIT_V1",
    256,
)

vit_b1024 = row_for(
    "STAGE21_MASKED_VIT_V1",
    1024,
)

ft_b64 = row_for(
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    64,
)

ft_b256 = row_for(
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    256,
)

ft_b1024 = row_for(
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    1024,
)


# -------------------------------
# GPU memory
# -------------------------------

gpu_pass = gpu[
    gpu["status"] == "PASS"
].copy()


def max_gpu_peak(target):
    z = gpu_pass[
        gpu_pass["target_id"] == target
    ].copy()

    z = z.dropna(
        subset=[
            "delta_peak_gpu_memory_mib"
        ]
    )

    if z.empty:
        raise RuntimeError(
            f"No GPU memory data for {target}"
        )

    idx = z[
        "delta_peak_gpu_memory_mib"
    ].idxmax()

    return z.loc[idx]


vit_mem = max_gpu_peak(
    "STAGE21_MASKED_VIT_V1"
)

cnn_mem = max_gpu_peak(
    "STAGE20_MASKED_CNN_V1"
)

ft_mem = max_gpu_peak(
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING"
)

xgb_mem = max_gpu_peak(
    "STAGE16_XGBOOST_TUNED"
)

cat_mem = max_gpu_peak(
    "STAGE16_CATBOOST_TUNED"
)


# -------------------------------
# Pareto
# -------------------------------

pareto_map = {
    r["target_id"]: r
    for _, r in pareto.iterrows()
}


lgb_p = pareto_map[
    "STAGE16_LIGHTGBM_TUNED"
]

xgb_p = pareto_map[
    "STAGE16_XGBOOST_TUNED"
]

cat_p = pareto_map[
    "STAGE16_CATBOOST_TUNED"
]

ft_p = pareto_map[
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING"
]

cnn_p = pareto_map[
    "STAGE20_MASKED_CNN_V1"
]

vit_p = pareto_map[
    "STAGE21_MASKED_VIT_V1"
]


print(
    "Batch-1 CNN GPU p95 speedup :",
    f2(
        cnn[
            "p95_latency_speedup_cpu_over_gpu"
        ]
    ),
    "x"
)

print(
    "Batch-1 FT GPU p95 speedup  :",
    f2(
        ft[
            "p95_latency_speedup_cpu_over_gpu"
        ]
    ),
    "x"
)

print(
    "Batch-1 XGB CPU advantage   :",
    f2(xgb_cpu_adv),
    "x"
)

print(
    "Batch-1 Cat CPU advantage   :",
    f2(cat_cpu_adv),
    "x"
)

print(
    "Batch-1 ViT ratio           :",
    f3(
        vit[
            "p95_latency_speedup_cpu_over_gpu"
        ]
    )
)


# =============================================================================
# 4. BUILD CLAIM TRACEABILITY
# =============================================================================

banner("STAGE26-P1 :: BUILD CLAIM TRACEABILITY")

trace = []


def add_trace(
    claim_id,
    section,
    claim,
    source,
    metrics,
    boundary,
):

    trace.append(
        {
            "claim_id":
                claim_id,

            "section":
                section,

            "claim":
                claim,

            "source_artifact":
                source,

            "source_metrics":
                metrics,

            "scientific_boundary":
                boundary,
        }
    )


boundary_component = (
    "COMPONENT_LEVEL_PREPARED_INPUT_TO_MATERIALIZED_PROBABILITY"
)


add_trace(
    "P1-C01",
    "Deployment Efficiency",
    (
        "Masked CNN batch-1 GPU p95 latency advantage."
    ),
    "T26_CPU1_GPU_BATCH1_ANCHOR.csv",
    (
        f"CPU1 p95={f6(cnn['cpu_p95_latency_ms'])} ms; "
        f"GPU p95={f6(cnn['gpu_p95_latency_ms'])} ms; "
        f"ratio={f3(cnn['p95_latency_speedup_cpu_over_gpu'])}"
    ),
    boundary_component,
)


add_trace(
    "P1-C02",
    "Deployment Efficiency",
    (
        "FT ensemble batch-1 GPU p95 latency advantage."
    ),
    "T26_CPU1_GPU_BATCH1_ANCHOR.csv",
    (
        f"CPU1 p95={f6(ft['cpu_p95_latency_ms'])} ms; "
        f"GPU p95={f6(ft['gpu_p95_latency_ms'])} ms; "
        f"ratio={f3(ft['p95_latency_speedup_cpu_over_gpu'])}"
    ),
    boundary_component,
)


add_trace(
    "P1-C03",
    "Deployment Efficiency",
    (
        "XGBoost batch-dependent CPU/GPU crossover."
    ),
    "T26_CPU1_GPU_MATCHED_COMPARISON.csv",
    (
        f"B1 ratio={f3(xgb['p95_latency_speedup_cpu_over_gpu'])}; "
        f"B256={f3(xgb_b256['p95_latency_speedup_cpu_over_gpu'])}; "
        f"B1024={f3(xgb_b1024['p95_latency_speedup_cpu_over_gpu'])}; "
        f"B8192={f3(xgb_b8192['p95_latency_speedup_cpu_over_gpu'])}"
    ),
    boundary_component,
)


add_trace(
    "P1-C04",
    "Deployment Efficiency",
    (
        "CatBoost remains CPU-favorable until very large batches."
    ),
    "T26_CPU1_GPU_MATCHED_COMPARISON.csv",
    (
        f"B1 ratio={f3(cat['p95_latency_speedup_cpu_over_gpu'])}; "
        f"B1024={f3(cat_b1024['p95_latency_speedup_cpu_over_gpu'])}; "
        f"B8192={f3(cat_b8192['p95_latency_speedup_cpu_over_gpu'])}"
    ),
    boundary_component,
)


add_trace(
    "P1-C05",
    "Resource Feasibility",
    (
        "CNN B8192 produced a genuine CUDA OOM."
    ),
    "T26_GPU_WARM_INFERENCE_MEMORY.csv",
    (
        "STAGE20_MASKED_CNN_V1, batch_size=8192, "
        "status=RESOURCE_LIMIT_OOM"
    ),
    "NON_PASS_COST_NOT_IMPUTED",
)


add_trace(
    "P1-C06",
    "Backend Availability",
    (
        "LightGBM and dependent ensemble GPU paths are unavailable."
    ),
    "T26_GPU_BACKEND_RESOURCE_STATUS.csv",
    (
        "LightGBM=BACKEND_UNAVAILABLE; "
        "ENS_LGBM_XGB_EQUAL=BACKEND_UNAVAILABLE"
    ),
    "NO_FORCED_BACKEND_SUBSTITUTION",
)


add_trace(
    "P1-C07",
    "Pareto Discussion",
    (
        "Group-A CPU frontier includes XGBoost, LightGBM, and CatBoost."
    ),
    "T26_PARETO.csv",
    (
        "XGBoost=FRONTIER_MEMBER; "
        "LightGBM=FRONTIER_MEMBER; "
        "CatBoost=FRONTIER_MEMBER"
    ),
    "WITHIN_GROUP_ONLY",
)


add_trace(
    "P1-C08",
    "Pareto Discussion",
    (
        "Masked ViT dominates masked CNN on the descriptive Group-B CPU frontier."
    ),
    "T26_PARETO.csv",
    (
        f"ViT PR-AUC={f6(vit_p['pr_auc'])}, "
        f"p95={f6(vit_p['cpu1_batch1_p95_latency_ms'])} ms; "
        f"CNN PR-AUC={f6(cnn_p['pr_auc'])}, "
        f"p95={f6(cnn_p['cpu1_batch1_p95_latency_ms'])} ms"
    ),
    "GROUP_B_ONLY",
)


trace_df = pd.DataFrame(
    trace
)

print(
    trace_df[
        [
            "claim_id",
            "section",
            "source_artifact",
        ]
    ].to_string(
        index=False
    )
)


# =============================================================================
# 5. BUILD MANUSCRIPT ASSET PLAN
# =============================================================================

banner("STAGE26-P1 :: BUILD IEEE ASSET PLAN")

asset_plan = pd.DataFrame(
    [
        {
            "order": 1,
            "asset":
                "F26_CPU1_GPU_P95_SPEEDUP",
            "placement":
                "MAIN_TEXT",
            "recommended_role":
                "Primary deployment-efficiency figure",
        },

        {
            "order": 2,
            "asset":
                "T26_CPU1_GPU_BATCH1_ANCHOR",
            "placement":
                "MAIN_TEXT",
            "recommended_role":
                "Low-batch online inference comparison",
        },

        {
            "order": 3,
            "asset":
                "F26_PARETO",
            "placement":
                "MAIN_TEXT",
            "recommended_role":
                "Within-group predictive-performance versus CPU latency trade-off",
        },

        {
            "order": 4,
            "asset":
                "F26_GPU_DELTA_PEAK_MEMORY",
            "placement":
                "MAIN_TEXT_OR_APPENDIX_IF_SPACE_LIMITED",
            "recommended_role":
                "GPU memory scaling and resource-limit context",
        },

        {
            "order": 5,
            "asset":
                "T26_CPU1_GPU_MATCHED_COMPARISON",
            "placement":
                "SUPPLEMENT",
            "recommended_role":
                "Full batchwise CPU/GPU numerical comparison",
        },

        {
            "order": 6,
            "asset":
                "T26_GPU_WARM_INFERENCE_MEMORY",
            "placement":
                "SUPPLEMENT",
            "recommended_role":
                "Complete GPU latency/memory profile",
        },

        {
            "order": 7,
            "asset":
                "T26_GPU_BACKEND_RESOURCE_STATUS",
            "placement":
                "SUPPLEMENT",
            "recommended_role":
                "Backend availability and resource-limit outcomes",
        },

        {
            "order": 8,
            "asset":
                "F26_COMPONENT_BOUNDARY",
            "placement":
                "METHODS_OR_SUPPLEMENT",
            "recommended_role":
                "Clarify extraction, representation, inference, and unavailable E2E boundary",
        },
    ]
)


print(
    asset_plan.to_string(
        index=False
    )
)


# =============================================================================
# 6. MANUSCRIPT PROSE
# =============================================================================

banner("STAGE26-P1 :: GENERATE IEEE-STYLE PROSE")

draft = f"""
# Stage 26 Manuscript Draft
## Deployment-Efficiency Results, Discussion, and Limitations

> Draft generated exclusively from frozen Stage26 publication artifacts at
> commit `{ANCHOR}`. No new inference, timing, memory profiling, model fitting,
> or corpus access was performed.

---

## A. Deployment Profiling Protocol and Claim Boundary

Deployment cost was evaluated separately from predictive discrimination.
Warm inference measurements used frozen model artifacts and prepared model
inputs, with the primary inference boundary defined as the interval from
prepared model input to a materialized attack-probability output. CPU profiling
was performed under fixed one- and two-physical-core conditions, while GPU
profiling used a prospectively selected single NVIDIA Tesla T4. GPU timing used
device synchronization before and after every timed region. Timing and memory
measurements were executed separately.

The deployment analysis is therefore **component-level rather than complete
end-to-end**. Raw packet extraction, representation construction, and model
inference were not combined into an additive full-pipeline latency estimate.
Accordingly, the results below should not be interpreted as measurements of
complete IDS throughput.

---

## B. CPU Accuracy–Latency Trade-off

Within the duplicate-safe 70-feature comparison group, the descriptive CPU
point-estimate frontier contained XGBoost, LightGBM, and CatBoost. LightGBM
provided the highest frozen PR-AUC ({f6(lgb_p['pr_auc'])}), followed closely by
XGBoost ({f6(xgb_p['pr_auc'])}) and CatBoost ({f6(cat_p['pr_auc'])}). Their
batch-1 CPU1 p95 component-level latencies were
{f6(lgb_p['cpu1_batch1_p95_latency_ms'])} ms,
{f6(xgb_p['cpu1_batch1_p95_latency_ms'])} ms, and
{f6(cat_p['cpu1_batch1_p95_latency_ms'])} ms, respectively.

The five-checkpoint FT Transformer ensemble achieved a lower frozen PR-AUC
({f6(ft_p['pr_auc'])}) and a substantially higher batch-1 CPU1 p95 latency
({f6(ft_p['cpu1_batch1_p95_latency_ms'])} ms); it was therefore not a member
of the Group-A descriptive frontier.

The packet-image models were evaluated only within their separate Group-B
comparison population. In this group, the masked ViT achieved both higher
frozen PR-AUC ({f6(vit_p['pr_auc'])}) and lower batch-1 CPU1 p95 latency
({f6(vit_p['cpu1_batch1_p95_latency_ms'])} ms) than the masked CNN
(PR-AUC {f6(cnn_p['pr_auc'])}; p95
{f6(cnn_p['cpu1_batch1_p95_latency_ms'])} ms). Thus, the masked ViT was the
Group-B descriptive frontier member, whereas the masked CNN was not.
No cross-group Pareto comparison is made.

---

## C. Batch-1 CPU-versus-GPU Inference

The batch-1 results show that GPU execution was **not universally beneficial**.

The largest immediate acceleration was observed for the masked CNN. Its p95
latency decreased from {f6(cnn['cpu_p95_latency_ms'])} ms on CPU1 to
{f6(cnn['gpu_p95_latency_ms'])} ms on the T4, corresponding to a
{f2(cnn['p95_latency_speedup_cpu_over_gpu'])}x point-estimate GPU advantage.
Median throughput increased from
{f2(cnn['cpu_median_throughput_samples_per_s'])} to
{f2(cnn['gpu_median_throughput_samples_per_s'])} flows/s.

The five-checkpoint FT Transformer ensemble also benefited at batch size 1.
Its p95 latency decreased from {f6(ft['cpu_p95_latency_ms'])} ms to
{f6(ft['gpu_p95_latency_ms'])} ms, a
{f2(ft['p95_latency_speedup_cpu_over_gpu'])}x point-estimate improvement.

The masked ViT was close to parity at batch size 1. CPU1 p95 latency was
{f6(vit['cpu_p95_latency_ms'])} ms and GPU p95 latency was
{f6(vit['gpu_p95_latency_ms'])} ms, yielding only a
{f3(vit['p95_latency_speedup_cpu_over_gpu'])}x point-estimate GPU advantage.

In contrast, the two profiled tree models favored CPU execution at batch size
1. XGBoost recorded {f6(xgb['cpu_p95_latency_ms'])} ms on CPU1 versus
{f6(xgb['gpu_p95_latency_ms'])} ms on the T4, making CPU1 approximately
{f2(xgb_cpu_adv)}x faster at the p95 point estimate. CatBoost showed an even
larger low-batch CPU advantage: {f6(cat['cpu_p95_latency_ms'])} ms on CPU1
versus {f6(cat['gpu_p95_latency_ms'])} ms on the T4, equivalent to
approximately {f2(cat_cpu_adv)}x lower p95 latency on CPU1.

These results indicate that accelerator selection for online IDS inference
cannot be reduced to a universal CPU-versus-GPU rule.

---

## D. Batch-Dependent Hardware Crossover

The batchwise measurements reveal distinct hardware-scaling regimes.

XGBoost changed from a CPU-favorable regime at small batches to a GPU-favorable
regime as batching increased. Its CPU1/GPU p95 ratio was
{f3(xgb['p95_latency_speedup_cpu_over_gpu'])} at B=1 and remained below 1 at
B=64, but reached {f3(xgb_b256['p95_latency_speedup_cpu_over_gpu'])} at B=256,
{f3(xgb_b1024['p95_latency_speedup_cpu_over_gpu'])} at B=1024, and
{f3(xgb_b8192['p95_latency_speedup_cpu_over_gpu'])} at B=8192. The observed
point-estimate crossover therefore occurred between the matched B=64 and B=256
conditions.

CatBoost crossed later. Its p95 CPU1/GPU ratio remained
{f3(cat_b1024['p95_latency_speedup_cpu_over_gpu'])} at B=1024 and increased to
{f3(cat_b8192['p95_latency_speedup_cpu_over_gpu'])} at B=8192. Thus, for the
measured conditions, GPU execution became favorable only at the largest batch.

The neural models exhibited a different pattern. The FT ensemble was already
GPU-favorable at B=1 and its p95 advantage expanded to
{f2(ft_b64['p95_latency_speedup_cpu_over_gpu'])}x at B=64,
{f2(ft_b256['p95_latency_speedup_cpu_over_gpu'])}x at B=256, and
{f2(ft_b1024['p95_latency_speedup_cpu_over_gpu'])}x at B=1024.

The masked CNN similarly showed a strong GPU advantage across every matched
CPU/GPU condition, rising from
{f2(cnn['p95_latency_speedup_cpu_over_gpu'])}x at B=1 to
{f2(cnn_b64['p95_latency_speedup_cpu_over_gpu'])}x at B=64 and
{f2(cnn_b256['p95_latency_speedup_cpu_over_gpu'])}x at B=256.

The masked ViT moved from near parity at B=1 to clearly GPU-favorable operation
at larger batches, with point-estimate p95 ratios of
{f2(vit_b64['p95_latency_speedup_cpu_over_gpu'])}x,
{f2(vit_b256['p95_latency_speedup_cpu_over_gpu'])}x, and
{f2(vit_b1024['p95_latency_speedup_cpu_over_gpu'])}x at B=64, B=256, and
B=1024, respectively.

These ratios are descriptive point estimates. Confidence intervals for derived
CPU/GPU ratios were not computed and should not be inferred from the separate
latency confidence intervals.

---

## E. GPU Memory and Resource Feasibility

GPU acceleration introduced architecture-specific memory constraints.

The masked ViT reached a maximum observed delta peak GPU process memory of
approximately {f2(vit_mem['delta_peak_gpu_memory_mib'])} MiB at B=
{int(vit_mem['batch_size'])}, while remaining measurable at B=8192.

The masked CNN reached approximately
{f2(cnn_mem['delta_peak_gpu_memory_mib'])} MiB of delta peak GPU process memory
at its largest successful condition, B={int(cnn_mem['batch_size'])}. At B=8192
the CNN failed with a genuine CUDA out-of-memory resource-limit outcome. This
condition was retained as `RESOURCE_LIMIT_OOM`; no latency, throughput, or
memory value was imputed.

The five-checkpoint FT ensemble reached approximately
{f2(ft_mem['delta_peak_gpu_memory_mib'])} MiB of maximum observed delta peak
GPU process memory. In comparison, the tree models required much smaller
incremental GPU process memory in the measured conditions: approximately
{f2(xgb_mem['delta_peak_gpu_memory_mib'])} MiB for XGBoost and
{f2(cat_mem['delta_peak_gpu_memory_mib'])} MiB for CatBoost.

Memory feasibility therefore constitutes a separate deployment dimension from
latency acceleration. A configuration with high GPU throughput can still be
constrained by accelerator-memory capacity at large batches.

---

## F. Backend Availability as a Deployment Constraint

Not every frozen model exposed a contract-approved native GPU inference path.

The frozen LightGBM deployment artifact did not provide a native GPU inference
route under the Stage26 no-conversion/no-substitution policy. LightGBM GPU
conditions were therefore recorded as `BACKEND_UNAVAILABLE`, rather than being
executed through CPU prediction and mislabeled as GPU measurements.

Because the operational LightGBM+XGBoost ensemble requires probability outputs
from both frozen members, the absence of a valid LightGBM GPU inference path
also made the complete ensemble GPU backend unavailable.

This distinction is important: `BACKEND_UNAVAILABLE` describes deployment-path
availability and should not be interpreted as evidence that LightGBM GPU
inference is slower than CPU inference.

---

## G. Deployment Implications

The combined CPU/GPU profile supports three practical observations.

First, **low-latency online inference and high-throughput batched inference can
favor different hardware choices**. At B=1, CPU1 was substantially faster for
XGBoost and CatBoost, whereas GPU execution strongly benefited the masked CNN
and materially benefited the FT ensemble.

Second, **the point at which GPU execution becomes advantageous is
architecture-dependent**. XGBoost crossed into a GPU-favorable regime by
B=256, whereas CatBoost did not do so until B=8192 in the measured conditions.
The neural architectures generally benefited from GPU execution much earlier.

Third, **deployment selection should jointly consider predictive performance,
latency, batching behavior, backend availability, and memory requirements**.
For example, the masked CNN achieved strong GPU acceleration but had lower
frozen predictive discrimination than the masked ViT within Group B and reached
a GPU memory limit at B=8192. Conversely, the Group-A tree models combined
strong frozen PR-AUC with very low CPU batch-1 latency, making CPU deployment
competitive for low-batch operation even when GPU acceleration became useful
at larger batches.

These observations argue against reporting a single hardware-independent
"fastest model." Instead, the appropriate deployment choice depends on the
operational regime and the scientifically comparable model group.

---

## H. Limitations of the Deployment Study

Several boundaries should constrain interpretation.

1. The measurements represent **component-level inference**, not a complete
   end-to-end IDS pipeline.

2. A scientifically compatible complete extraction-to-inference measurement
   was unavailable. Raw extraction, representation construction, and isolated
   inference measurements must therefore remain separate.

3. The duplicate-safe 70-feature models and packet-image models were evaluated
   on different scientifically defined comparison populations; Pareto
   frontiers must remain within their respective groups.

4. CPU/GPU speedup ratios are descriptive point estimates. Ratio-level
   confidence intervals were not computed.

5. Resource-limit outcomes and unavailable backends were retained explicitly;
   missing deployment costs were not imputed.

6. Batch sizes were frozen before profiling. No post-hoc "best batch" or
   "optimal batch size" claim is made.

7. GPU results were obtained on a single Tesla T4 deployment profile. They
   should not be generalized to all GPU architectures without additional
   hardware-specific measurement.

---

## I. Recommended Main-Paper Presentation

### Main deployment figure
**F26_CPU1_GPU_P95_SPEEDUP**

Recommended message:
> GPU acceleration is architecture- and batch-dependent rather than universal.

### Main deployment table
**T26_CPU1_GPU_BATCH1_ANCHOR**

Recommended message:
> Low-batch online deployment can favor CPU inference for tree models while
> GPU execution benefits packet-image and transformer models.

### Accuracy–cost figure
**F26_PARETO**

Use separate Group-A and Group-B panels only.

Recommended message:
> Predictive discrimination and deployment cost must be interpreted within
> scientifically comparable representation groups.

### GPU resource figure
**F26_GPU_DELTA_PEAK_MEMORY**

Recommended message:
> Accelerator memory becomes a practical deployment constraint for large
> packet-image batches.

---

## J. Suggested Section-Level Takeaway

The principal deployment finding is not that GPU execution is universally
faster. Rather, the frozen models exhibit **architecture-dependent and
batch-dependent hardware regimes**. Tree models provide extremely low
single-flow CPU latency, while neural architectures derive much larger benefits
from GPU parallelism. Consequently, deployment decisions for an IDS should be
based on the expected batching regime, model representation, predictive
performance, backend availability, and memory budget rather than on model
accuracy or accelerator availability alone.
""".strip() + "\n"


# =============================================================================
# 7. WRITE TEMPORARY REVIEW ARTIFACTS
# =============================================================================

banner("STAGE26-P1 :: WRITE REVIEW ARTIFACTS")

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


draft_path = (
    OUT
    / "stage26_results_deployment_discussion_draft.md"
)

trace_path = (
    OUT
    / "stage26_claim_traceability.csv"
)

asset_path = (
    OUT
    / "stage26_manuscript_asset_plan.csv"
)


draft_path.write_text(
    draft,
    encoding="utf-8",
)

trace_df.to_csv(
    trace_path,
    index=False,
)

asset_plan.to_csv(
    asset_path,
    index=False,
)


for path in [
    draft_path,
    trace_path,
    asset_path,
]:

    print()
    print(path.name)
    print(
        "  bytes :",
        path.stat().st_size,
    )
    print(
        "  SHA256:",
        sha256_file(path),
    )


# =============================================================================
# 8. DISPLAY DRAFT
# =============================================================================

banner("STAGE26-P1 :: MANUSCRIPT DRAFT PREVIEW")

print(draft)


# =============================================================================
# 9. FINAL SAFETY CHECK
# =============================================================================

banner("STAGE26-P1 COMPLETE")

final_status = git(
    "status",
    "--porcelain",
)

print(
    "Final Stage26 anchor    :",
    ANCHOR,
)

print(
    "New measurements        : NONE"
)

print(
    "Inference/model loading : NONE"
)

print(
    "Corpus access           : NONE"
)

print(
    "Ratio CIs invented      : NO"
)

print(
    "Cross-group Pareto      : NO"
)

print(
    "Missing costs imputed   : NO"
)

print(
    "Repository modified     :",
    bool(final_status),
)

print()
print(
    "Draft                   :",
    draft_path,
)

print(
    "Traceability            :",
    trace_path,
)

print(
    "Asset plan              :",
    asset_path,
)

print()
print(
    "NEXT:"
)

print(
    "  Paste the COMPLETE output."
)

print(
    "  Then we will review the prose scientifically, tighten it for IEEE length, "
    "and decide whether to freeze/commit it as the paper-facing Stage26 narrative."
)


if final_status:
    raise RuntimeError(
        "Repository unexpectedly changed during P1."
    )


STAGE26-P1 :: FROZEN REPOSITORY GATE
Expected anchor : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
Local HEAD      : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
origin/main     : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
Repo clean      : True

STAGE26-P1 :: LOAD PUBLICATION SOURCES
Matched CPU/GPU rows : 26
B1 rows              : 6
GPU rows             : 40
GPU status rows      : 8
CPU Pareto rows      : 6
CPU warm rows        : 80
P0 claim rows        : 7

STAGE26-P1 :: RESOLVE MANUSCRIPT VALUES
Batch-1 CNN GPU p95 speedup : 10.86 x
Batch-1 FT GPU p95 speedup  : 1.94 x
Batch-1 XGB CPU advantage   : 3.86 x
Batch-1 Cat CPU advantage   : 6.18 x
Batch-1 ViT ratio           : 1.048

STAGE26-P1 :: BUILD CLAIM TRACEABILITY
claim_id               section                     source_artifact
  P1-C01 Deployment Efficiency      T26_CPU1_GPU_BATCH1_ANCHOR.csv
  P1-C02 Deployment Efficiency      T26_CPU1_GPU_BATCH1_ANCHOR.csv
  P1-C03 Deployment Efficiency T26_CPU1_GPU_MATCHED_COMPARISON.csv
  P1-C04 D

In [6]:
# =============================================================================
# STAGE26-P2 :: FREEZE IEEE-READY PAPER-FACING DEPLOYMENT NARRATIVE
#
# PURPOSE
#   - Re-derive a tightened IEEE-ready narrative directly from frozen Stage26
#     publication artifacts.
#   - Preserve exact claim boundaries and traceability.
#   - Commit/push the derived narrative as a reproducible manuscript-support
#     artifact.
#
# NO inference
# NO timing
# NO model loading
# NO corpus access
# NO new scientific measurement
# NO modification of frozen Stage26 measurement artifacts
# =============================================================================

from __future__ import annotations

import csv
import hashlib
import json
import os
import stat
import subprocess
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# =============================================================================
# FROZEN IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "9e8354ecc9cfa72c28aa037e5d2053de422bf7a2"
)

G2 = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_g2_final_gpu_publication_closure"
)

CPU_TABLES = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_8d3_cpu_publication_tables"
)

OUT = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
    / "stage26_p2_paper_facing_narrative"
)

COMMIT_MSG = (
    "stage26: freeze paper-facing deployment narrative"
)

DISPLAY = {
    "STAGE16_XGBOOST_TUNED":
        "XGBoost",

    "STAGE16_LIGHTGBM_TUNED":
        "LightGBM",

    "STAGE16_CATBOOST_TUNED":
        "CatBoost",

    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING":
        "FT Transformer ensemble",

    "STAGE20_MASKED_CNN_V1":
        "masked CNN",

    "STAGE21_MASKED_VIT_V1":
        "masked ViT",

    "ENS_LGBM_XGB_EQUAL":
        "LightGBM+XGBoost ensemble",

    "FT_BALANCED_SINGLE_RESOURCE_REFERENCE":
        "single-checkpoint FT reference",
}


# =============================================================================
# HELPERS
# =============================================================================

def banner(text):
    print()
    print("=" * 122)
    print(text)
    print("=" * 122)


def run(cmd, *, cwd=None, env=None, check=True):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(map(str, cmd))}\n\n"
            f"{p.stdout}"
        )

    return p.stdout.strip()


def git(*args, env=None, check=True):
    return run(
        ["git", *args],
        cwd=REPO,
        env=env,
        check=check,
    )


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for block in iter(
            lambda: f.read(16 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def f2(x):
    return f"{float(x):.2f}"


def f3(x):
    return f"{float(x):.3f}"


def f6(x):
    return f"{float(x):.6f}"


def reciprocal(x):
    x = float(x)

    if x <= 0:
        raise ValueError(
            "Cannot take reciprocal of non-positive ratio."
        )

    return 1.0 / x


def one_row(df, target, batch=None):
    z = df[
        df["target_id"] == target
    ]

    if batch is not None:
        z = z[
            z["batch_size"] == batch
        ]

    if len(z) != 1:
        raise RuntimeError(
            f"Expected exactly one row for {target}, "
            f"batch={batch}; got {len(z)}"
        )

    return z.iloc[0]


# =============================================================================
# 1. DURABLE PARENT GATE
# =============================================================================

banner("STAGE26-P2 :: DURABLE PARENT GATE")

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)

print("Expected parent :", EXPECTED_PARENT)
print("Local HEAD      :", head)
print("origin/main     :", origin)
print("Repo clean      :", status == "")

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Unexpected local HEAD."
    )

if origin != EXPECTED_PARENT:
    raise RuntimeError(
        "Unexpected origin/main."
    )

if status:
    raise RuntimeError(
        "Repository must be clean before P2."
    )

if OUT.exists():
    raise RuntimeError(
        f"P2 output already exists: {OUT}"
    )


# =============================================================================
# 2. LOAD FROZEN SOURCES
# =============================================================================

banner("STAGE26-P2 :: LOAD FROZEN PUBLICATION SOURCES")

source_paths = {
    "gpu_warm":
        G2
        / "T26_GPU_WARM_INFERENCE_MEMORY.csv",

    "matched":
        G2
        / "T26_CPU1_GPU_MATCHED_COMPARISON.csv",

    "gpu_status":
        G2
        / "T26_GPU_BACKEND_RESOURCE_STATUS.csv",

    "batch1":
        G2
        / "T26_CPU1_GPU_BATCH1_ANCHOR.csv",

    "pareto":
        CPU_TABLES
        / "T26_PARETO.csv",

    "cpu_warm":
        CPU_TABLES
        / "T26_CPU_WARM_INFERENCE.csv",
}


for name, path in source_paths.items():

    if not path.is_file():
        raise FileNotFoundError(
            path
        )

    print(
        f"{name:12s} "
        f"{path.relative_to(REPO)}"
    )

    print(
        " " * 13
        + sha256_file(path)
    )


gpu = pd.read_csv(
    source_paths["gpu_warm"]
)

matched = pd.read_csv(
    source_paths["matched"]
)

gpu_status = pd.read_csv(
    source_paths["gpu_status"]
)

b1 = pd.read_csv(
    source_paths["batch1"]
)

pareto = pd.read_csv(
    source_paths["pareto"]
)

cpu_warm = pd.read_csv(
    source_paths["cpu_warm"]
)


print()
print("GPU rows             :", len(gpu))
print("Matched CPU/GPU rows :", len(matched))
print("GPU status rows      :", len(gpu_status))
print("Batch-1 rows         :", len(b1))
print("Pareto rows          :", len(pareto))
print("CPU warm rows        :", len(cpu_warm))


if len(gpu) != 40:
    raise RuntimeError(
        "Expected 40 frozen GPU conditions."
    )

if len(matched) != 26:
    raise RuntimeError(
        "Expected 26 matched CPU1/GPU PASS comparisons."
    )

if len(gpu_status) != 8:
    raise RuntimeError(
        "Expected 8 GPU target-status rows."
    )

if len(b1) != 6:
    raise RuntimeError(
        "Expected 6 batch-1 CPU/GPU rows."
    )

if len(pareto) != 6:
    raise RuntimeError(
        "Expected 6 CPU Pareto rows."
    )

if len(cpu_warm) != 80:
    raise RuntimeError(
        "Expected 80 frozen CPU warm rows."
    )


# =============================================================================
# 3. SCIENTIFIC GEOMETRY AUDIT
# =============================================================================

banner("STAGE26-P2 :: SCIENTIFIC GEOMETRY AUDIT")

status_counts = (
    gpu["status"]
    .value_counts()
    .to_dict()
)

print(
    "GPU status counts:",
    status_counts,
)

if status_counts != {
    "PASS": 29,
    "BACKEND_UNAVAILABLE": 10,
    "RESOURCE_LIMIT_OOM": 1,
}:
    raise RuntimeError(
        "Frozen GPU status geometry changed."
    )


raw_nonpass = gpu[
    gpu["status"] != "PASS"
].copy()


cost_cols = [
    "gpu_p50_latency_ms"
        if "gpu_p50_latency_ms" in gpu.columns
        else None,
]

# The G2 GPU table uses publication-schema names. Verify that non-PASS
# rows do not contain timing/memory measurements wherever those columns exist.
candidate_cost_cols = [
    c for c in gpu.columns
    if (
        "latency" in c.lower()
        or "throughput" in c.lower()
        or "delta_peak" in c.lower()
    )
]

for c in candidate_cost_cols:

    vals = raw_nonpass[c]

    populated = vals.notna()

    # String status/detail columns cannot appear because name filter above
    # selects numerical cost fields only.
    if populated.any():

        bad = raw_nonpass.loc[
            populated,
            [
                "target_id",
                "batch_size",
                "status",
                c,
            ],
        ]

        print(bad)

        raise RuntimeError(
            f"Non-PASS cost imputation detected in {c}"
        )


print("29 PASS / 10 backend / 1 OOM : PASS")
print("Non-PASS cost imputation     : NONE")
print("Complete E2E                 : UNAVAILABLE")
print("Cross-group Pareto           : PROHIBITED")
print("CPU/GPU ratio CIs            : NOT COMPUTED")


# =============================================================================
# 4. RESOLVE NUMERICAL EVIDENCE
# =============================================================================

banner("STAGE26-P2 :: RESOLVE PAPER-FACING EVIDENCE")

b1_map = {
    r["target_id"]: r
    for _, r in b1.iterrows()
}

pareto_map = {
    r["target_id"]: r
    for _, r in pareto.iterrows()
}


xgb = b1_map[
    "STAGE16_XGBOOST_TUNED"
]

cat = b1_map[
    "STAGE16_CATBOOST_TUNED"
]

ft = b1_map[
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING"
]

cnn = b1_map[
    "STAGE20_MASKED_CNN_V1"
]

vit = b1_map[
    "STAGE21_MASKED_VIT_V1"
]


xgb_p = pareto_map[
    "STAGE16_XGBOOST_TUNED"
]

lgb_p = pareto_map[
    "STAGE16_LIGHTGBM_TUNED"
]

cat_p = pareto_map[
    "STAGE16_CATBOOST_TUNED"
]

ft_p = pareto_map[
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING"
]

cnn_p = pareto_map[
    "STAGE20_MASKED_CNN_V1"
]

vit_p = pareto_map[
    "STAGE21_MASKED_VIT_V1"
]


xgb_b64 = one_row(
    matched,
    "STAGE16_XGBOOST_TUNED",
    64,
)

xgb_b256 = one_row(
    matched,
    "STAGE16_XGBOOST_TUNED",
    256,
)

xgb_b1024 = one_row(
    matched,
    "STAGE16_XGBOOST_TUNED",
    1024,
)

xgb_b8192 = one_row(
    matched,
    "STAGE16_XGBOOST_TUNED",
    8192,
)


cat_b1024 = one_row(
    matched,
    "STAGE16_CATBOOST_TUNED",
    1024,
)

cat_b8192 = one_row(
    matched,
    "STAGE16_CATBOOST_TUNED",
    8192,
)


ft_b64 = one_row(
    matched,
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    64,
)

ft_b256 = one_row(
    matched,
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    256,
)

ft_b1024 = one_row(
    matched,
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING",
    1024,
)


cnn_b64 = one_row(
    matched,
    "STAGE20_MASKED_CNN_V1",
    64,
)

cnn_b256 = one_row(
    matched,
    "STAGE20_MASKED_CNN_V1",
    256,
)


vit_b64 = one_row(
    matched,
    "STAGE21_MASKED_VIT_V1",
    64,
)

vit_b256 = one_row(
    matched,
    "STAGE21_MASKED_VIT_V1",
    256,
)

vit_b1024 = one_row(
    matched,
    "STAGE21_MASKED_VIT_V1",
    1024,
)


def max_memory_row(target):
    z = gpu[
        (gpu["target_id"] == target)
        &
        (gpu["status"] == "PASS")
    ].copy()

    z = z.dropna(
        subset=[
            "delta_peak_gpu_memory_mib"
        ]
    )

    if z.empty:
        raise RuntimeError(
            f"No PASS GPU memory rows for {target}"
        )

    i = z[
        "delta_peak_gpu_memory_mib"
    ].idxmax()

    return z.loc[i]


vit_mem = max_memory_row(
    "STAGE21_MASKED_VIT_V1"
)

cnn_mem = max_memory_row(
    "STAGE20_MASKED_CNN_V1"
)

ft_mem = max_memory_row(
    "FT_BALANCED_5_CHECKPOINT_SOFT_VOTING"
)

xgb_mem = max_memory_row(
    "STAGE16_XGBOOST_TUNED"
)

cat_mem = max_memory_row(
    "STAGE16_CATBOOST_TUNED"
)


cnn_oom = gpu[
    (gpu["target_id"] == "STAGE20_MASKED_CNN_V1")
    &
    (gpu["batch_size"] == 8192)
]

if len(cnn_oom) != 1:
    raise RuntimeError(
        "CNN B8192 geometry mismatch."
    )

if cnn_oom.iloc[0]["status"] != "RESOURCE_LIMIT_OOM":
    raise RuntimeError(
        "CNN B8192 is no longer RESOURCE_LIMIT_OOM."
    )


print(
    "CNN B1 p95 GPU ratio :",
    f3(
        cnn[
            "p95_latency_speedup_cpu_over_gpu"
        ]
    ),
)

print(
    "FT B1 p95 GPU ratio  :",
    f3(
        ft[
            "p95_latency_speedup_cpu_over_gpu"
        ]
    ),
)

print(
    "ViT B1 p95 ratio     :",
    f3(
        vit[
            "p95_latency_speedup_cpu_over_gpu"
        ]
    ),
)

print(
    "XGB B1 CPU advantage :",
    f3(
        reciprocal(
            xgb[
                "p95_latency_speedup_cpu_over_gpu"
            ]
        )
    ),
)

print(
    "Cat B1 CPU advantage :",
    f3(
        reciprocal(
            cat[
                "p95_latency_speedup_cpu_over_gpu"
            ]
        )
    ),
)

print(
    "XGB first measured GPU-favorable batch : 256"
)

print(
    "Cat first measured GPU-favorable batch : 8192"
)


# =============================================================================
# 5. BUILD TIGHTENED IEEE-READY NARRATIVE
# =============================================================================

banner("STAGE26-P2 :: BUILD TIGHTENED IEEE-READY NARRATIVE")


narrative = f"""
# Stage26 Paper-Facing Deployment Narrative

**Frozen scientific source:** `{EXPECTED_PARENT}`

**Status:** Derived manuscript-support artifact. This document does not replace
the frozen measurement tables and does not constitute a new experiment.

## Deployment Profiling Boundary

Deployment efficiency was evaluated independently of predictive discrimination.
Warm inference used frozen model artifacts and prepared model inputs, with the
timed boundary extending from prepared input to a materialized attack-probability
output. CPU profiling used fixed physical-core configurations, whereas GPU
profiling used a prospectively selected single NVIDIA Tesla T4 with device
synchronization before and after every timed region. Timing and memory runs were
separate. These measurements therefore describe **component-level inference**;
they are not complete extraction-to-decision IDS latency or throughput
measurements.

## Accuracy-Latency Context

Within the duplicate-safe 70-feature comparison group, XGBoost, LightGBM, and
CatBoost were descriptive CPU frontier members. Their frozen PR-AUC values were
{f6(xgb_p['pr_auc'])}, {f6(lgb_p['pr_auc'])}, and
{f6(cat_p['pr_auc'])}, respectively, with batch-1 CPU1 p95 inference
latencies of {f6(xgb_p['cpu1_batch1_p95_latency_ms'])},
{f6(lgb_p['cpu1_batch1_p95_latency_ms'])}, and
{f6(cat_p['cpu1_batch1_p95_latency_ms'])} ms. The five-checkpoint FT
Transformer ensemble had lower PR-AUC ({f6(ft_p['pr_auc'])}) and higher
batch-1 CPU1 p95 latency ({f6(ft_p['cpu1_batch1_p95_latency_ms'])} ms)
and was not a Group-A frontier member.

The packet-image models form a separate comparison group. Within Group B, the
masked ViT achieved higher PR-AUC ({f6(vit_p['pr_auc'])}) and lower batch-1
CPU1 p95 latency ({f6(vit_p['cpu1_batch1_p95_latency_ms'])} ms) than the
masked CNN (PR-AUC {f6(cnn_p['pr_auc'])}; p95
{f6(cnn_p['cpu1_batch1_p95_latency_ms'])} ms). The ViT was therefore the
descriptive Group-B frontier member. No cross-group Pareto comparison is made.

## CPU-GPU Deployment Behavior

GPU acceleration was architecture- and batch-dependent rather than universal.
At batch size 1, the masked CNN showed the largest immediate GPU benefit: p95
component-level latency decreased from {f6(cnn['cpu_p95_latency_ms'])} ms on
CPU1 to {f6(cnn['gpu_p95_latency_ms'])} ms on the T4, a
{f2(cnn['p95_latency_speedup_cpu_over_gpu'])}x point-estimate ratio. The FT
Transformer ensemble also benefited at B=1, decreasing from
{f6(ft['cpu_p95_latency_ms'])} to {f6(ft['gpu_p95_latency_ms'])} ms
({f2(ft['p95_latency_speedup_cpu_over_gpu'])}x). The masked ViT was close
to parity, with CPU1 and GPU p95 latencies of
{f6(vit['cpu_p95_latency_ms'])} and {f6(vit['gpu_p95_latency_ms'])} ms,
respectively ({f3(vit['p95_latency_speedup_cpu_over_gpu'])}x).

The profiled tree models showed the opposite low-batch pattern. XGBoost
recorded {f6(xgb['cpu_p95_latency_ms'])} ms on CPU1 and
{f6(xgb['gpu_p95_latency_ms'])} ms on the T4, corresponding to approximately
{f2(reciprocal(xgb['p95_latency_speedup_cpu_over_gpu']))}x lower p95
latency on CPU1. CatBoost recorded {f6(cat['cpu_p95_latency_ms'])} ms on
CPU1 versus {f6(cat['gpu_p95_latency_ms'])} ms on the T4, or approximately
{f2(reciprocal(cat['p95_latency_speedup_cpu_over_gpu']))}x lower p95
latency on CPU1.

Batching changed these relationships. XGBoost remained CPU-favorable at B=64
with a CPU1/GPU p95 ratio of
{f3(xgb_b64['p95_latency_speedup_cpu_over_gpu'])}, reached near crossover at
B=256 ({f3(xgb_b256['p95_latency_speedup_cpu_over_gpu'])}), and became
increasingly GPU-favorable at B=1024 and B=8192
({f3(xgb_b1024['p95_latency_speedup_cpu_over_gpu'])}x and
{f3(xgb_b8192['p95_latency_speedup_cpu_over_gpu'])}x). CatBoost crossed much
later: its ratio remained {f3(cat_b1024['p95_latency_speedup_cpu_over_gpu'])}
at B=1024 and reached
{f3(cat_b8192['p95_latency_speedup_cpu_over_gpu'])} at B=8192.

The neural models benefited earlier from parallel execution. The FT ensemble
reached point-estimate p95 ratios of
{f2(ft_b64['p95_latency_speedup_cpu_over_gpu'])}x,
{f2(ft_b256['p95_latency_speedup_cpu_over_gpu'])}x, and
{f2(ft_b1024['p95_latency_speedup_cpu_over_gpu'])}x at B=64, B=256, and
B=1024. The masked CNN reached
{f2(cnn_b64['p95_latency_speedup_cpu_over_gpu'])}x at B=64 and
{f2(cnn_b256['p95_latency_speedup_cpu_over_gpu'])}x at B=256. The masked
ViT moved from near parity at B=1 to
{f2(vit_b64['p95_latency_speedup_cpu_over_gpu'])}x,
{f2(vit_b256['p95_latency_speedup_cpu_over_gpu'])}x, and
{f2(vit_b1024['p95_latency_speedup_cpu_over_gpu'])}x at B=64, B=256, and
B=1024. These hardware ratios are descriptive point estimates; ratio-level
confidence intervals were not computed.

## Memory and Backend Constraints

GPU feasibility also depended on accelerator memory. The masked ViT remained
measurable at B=8192 while reaching approximately
{f2(vit_mem['delta_peak_gpu_memory_mib'])} MiB of maximum observed delta peak
GPU process memory. The masked CNN reached approximately
{f2(cnn_mem['delta_peak_gpu_memory_mib'])} MiB at its largest successful
condition, B={int(cnn_mem['batch_size'])}, and produced a genuine
`RESOURCE_LIMIT_OOM` at B=8192. The FT ensemble reached approximately
{f2(ft_mem['delta_peak_gpu_memory_mib'])} MiB, whereas XGBoost and CatBoost
required only approximately {f2(xgb_mem['delta_peak_gpu_memory_mib'])} MiB
and {f2(cat_mem['delta_peak_gpu_memory_mib'])} MiB of maximum observed delta
peak process memory, respectively.

The frozen LightGBM artifact did not expose a contract-approved native GPU
inference route under the no-conversion/no-substitution policy. Its GPU
conditions were therefore retained as `BACKEND_UNAVAILABLE`. Because the
operational LightGBM+XGBoost ensemble requires both constituent probability
outputs, that complete GPU ensemble path was also `BACKEND_UNAVAILABLE`.
Neither outcome should be interpreted as a latency comparison.

## Deployment Interpretation

The deployment results do not support a hardware-independent "fastest model."
Instead, they identify distinct operational regimes. The Group-A tree models
combine strong predictive discrimination with very low single-flow CPU latency,
making CPU execution attractive for low-batch online operation. XGBoost becomes
GPU-favorable as batching increases, whereas CatBoost crosses only at the
largest measured batch. The neural models exploit GPU parallelism much earlier,
but their deployment value must still be considered jointly with predictive
performance and memory demand. In Group B, for example, the masked CNN showed
stronger GPU acceleration but lower frozen PR-AUC than the masked ViT and
encountered an accelerator-memory ceiling at B=8192.

Accordingly, IDS deployment decisions should jointly consider model
representation, predictive performance, expected batching regime, backend
availability, latency requirements, and accelerator-memory budget rather than
accuracy or accelerator availability in isolation.

## Interpretation Boundaries

The following restrictions remain part of the frozen Stage26 interpretation:

1. Deployment measurements are component-level, not complete E2E IDS latency.
2. Group-A and Group-B Pareto results must remain separate.
3. CPU/GPU speedup ratios are point estimates; ratio CIs were not computed.
4. No missing deployment cost is imputed for OOM, timeout, or unavailable
   backends.
5. Frozen batch sizes are descriptive measurement conditions, not post-hoc
   optimized batch sizes.
6. The single-T4 measurements should not be generalized to other accelerator
   architectures without additional hardware-specific profiling.
""".strip() + "\n"


# =============================================================================
# 6. BUILD CLAIM TRACEABILITY
# =============================================================================

banner("STAGE26-P2 :: BUILD CLAIM TRACEABILITY")


trace_rows = [
    {
        "claim_id": "P2-C01",
        "topic": "Group A CPU frontier",
        "claim":
            "XGBoost, LightGBM, and CatBoost are descriptive "
            "Group-A CPU frontier members.",
        "source":
            "T26_PARETO.csv",
        "boundary":
            "WITHIN_GROUP_A_ONLY",
    },
    {
        "claim_id": "P2-C02",
        "topic": "Group B CPU frontier",
        "claim":
            "Masked ViT has higher PR-AUC and lower CPU1 B1 p95 "
            "latency than masked CNN within Group B.",
        "source":
            "T26_PARETO.csv",
        "boundary":
            "WITHIN_GROUP_B_ONLY",
    },
    {
        "claim_id": "P2-C03",
        "topic": "CNN B1 acceleration",
        "claim":
            f"CNN B1 CPU/GPU p95 point-estimate ratio="
            f"{f6(cnn['p95_latency_speedup_cpu_over_gpu'])}.",
        "source":
            "T26_CPU1_GPU_BATCH1_ANCHOR.csv",
        "boundary":
            "COMPONENT_LEVEL_POINT_ESTIMATE",
    },
    {
        "claim_id": "P2-C04",
        "topic": "FT B1 acceleration",
        "claim":
            f"FT ensemble B1 CPU/GPU p95 point-estimate ratio="
            f"{f6(ft['p95_latency_speedup_cpu_over_gpu'])}.",
        "source":
            "T26_CPU1_GPU_BATCH1_ANCHOR.csv",
        "boundary":
            "COMPONENT_LEVEL_POINT_ESTIMATE",
    },
    {
        "claim_id": "P2-C05",
        "topic": "XGBoost crossover",
        "claim":
            "XGBoost is CPU-favorable through B64 and GPU-favorable "
            "from measured B256 onward.",
        "source":
            "T26_CPU1_GPU_MATCHED_COMPARISON.csv",
        "boundary":
            "MEASURED_BATCHES_ONLY",
    },
    {
        "claim_id": "P2-C06",
        "topic": "CatBoost crossover",
        "claim":
            "CatBoost is still CPU-favorable at B1024 and becomes "
            "GPU-favorable at measured B8192.",
        "source":
            "T26_CPU1_GPU_MATCHED_COMPARISON.csv",
        "boundary":
            "MEASURED_BATCHES_ONLY",
    },
    {
        "claim_id": "P2-C07",
        "topic": "CNN resource limit",
        "claim":
            "CNN B8192 is RESOURCE_LIMIT_OOM with no imputed cost.",
        "source":
            "T26_GPU_WARM_INFERENCE_MEMORY.csv",
        "boundary":
            "RESOURCE_LIMIT_OUTCOME",
    },
    {
        "claim_id": "P2-C08",
        "topic": "LightGBM backend",
        "claim":
            "LightGBM native GPU inference and the dependent "
            "LightGBM+XGBoost complete GPU ensemble are "
            "BACKEND_UNAVAILABLE.",
        "source":
            "T26_GPU_BACKEND_RESOURCE_STATUS.csv",
        "boundary":
            "NO_BACKEND_SUBSTITUTION",
    },
]


trace_df = pd.DataFrame(
    trace_rows
)


print(
    trace_df[
        [
            "claim_id",
            "topic",
            "source",
            "boundary",
        ]
    ].to_string(
        index=False
    )
)


# =============================================================================
# 7. BUILD MANUSCRIPT ASSET MAP
# =============================================================================

banner("STAGE26-P2 :: BUILD MANUSCRIPT ASSET MAP")


asset_rows = [
    {
        "priority": 1,
        "asset":
            "F26_CPU1_GPU_P95_SPEEDUP",
        "placement":
            "MAIN_TEXT",
        "purpose":
            "Primary architecture/batch-dependent acceleration figure",
    },
    {
        "priority": 2,
        "asset":
            "T26_CPU1_GPU_BATCH1_ANCHOR",
        "placement":
            "MAIN_TEXT",
        "purpose":
            "Low-batch online deployment anchor",
    },
    {
        "priority": 3,
        "asset":
            "F26_PARETO",
        "placement":
            "MAIN_TEXT",
        "purpose":
            "Within-group predictive-performance/CPU-latency trade-off",
    },
    {
        "priority": 4,
        "asset":
            "F26_GPU_DELTA_PEAK_MEMORY",
        "placement":
            "MAIN_TEXT_IF_SPACE__OTHERWISE_SUPPLEMENT",
        "purpose":
            "GPU memory scaling and CNN resource ceiling",
    },
    {
        "priority": 5,
        "asset":
            "T26_CPU1_GPU_MATCHED_COMPARISON",
        "placement":
            "SUPPLEMENT",
        "purpose":
            "Complete matched CPU1/GPU batchwise point estimates",
    },
    {
        "priority": 6,
        "asset":
            "T26_GPU_WARM_INFERENCE_MEMORY",
        "placement":
            "SUPPLEMENT",
        "purpose":
            "Complete GPU latency/memory profile",
    },
    {
        "priority": 7,
        "asset":
            "T26_GPU_BACKEND_RESOURCE_STATUS",
        "placement":
            "SUPPLEMENT",
        "purpose":
            "Backend and resource-limit disclosure",
    },
    {
        "priority": 8,
        "asset":
            "F26_COMPONENT_BOUNDARY",
        "placement":
            "METHODS_OR_SUPPLEMENT",
        "purpose":
            "Explicit component/E2E measurement boundary",
    },
]


asset_df = pd.DataFrame(
    asset_rows
)

print(
    asset_df.to_string(
        index=False
    )
)


# =============================================================================
# 8. WRITE P2 PACKAGE
# =============================================================================

banner("STAGE26-P2 :: WRITE NARRATIVE PACKAGE")

OUT.mkdir(
    parents=True,
    exist_ok=False,
)


narrative_path = (
    OUT
    / "stage26_deployment_ieee_ready_narrative.md"
)

trace_path = (
    OUT
    / "stage26_deployment_claim_traceability.csv"
)

asset_path = (
    OUT
    / "stage26_deployment_manuscript_asset_map.csv"
)

receipt_path = (
    OUT
    / "stage26_p2_narrative_receipt.json"
)

manifest_path = (
    OUT
    / "stage26_p2_narrative_manifest.json"
)


narrative_path.write_text(
    narrative,
    encoding="utf-8",
)

trace_df.to_csv(
    trace_path,
    index=False,
)

asset_df.to_csv(
    asset_path,
    index=False,
)


source_hashes = {
    str(path.relative_to(REPO)):
        sha256_file(path)
    for path in source_paths.values()
}


receipt = {
    "schema":
        "stage26_p2_paper_facing_narrative_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent":
        EXPECTED_PARENT,

    "artifact_role":
        "DERIVED_MANUSCRIPT_SUPPORT_NOT_NEW_SCIENCE",

    "source_policy":
        "FROZEN_STAGE26_PUBLICATION_ARTIFACTS_ONLY",

    "source_hashes":
        source_hashes,

    "scientific_geometry": {
        "gpu_conditions": 40,
        "gpu_pass": 29,
        "gpu_backend_unavailable": 10,
        "gpu_resource_limit_oom": 1,
        "matched_cpu1_gpu_pass_rows": 26,
        "batch1_anchor_rows": 6,
    },

    "claim_boundaries": {
        "component_level_only": True,
        "complete_e2e_available": False,
        "cross_group_pareto_allowed": False,
        "ratio_confidence_intervals_computed": False,
        "missing_cost_imputation": False,
        "post_hoc_best_batch": False,
    },

    "new_measurement": False,
    "inference_performed": False,
    "model_loaded": False,
    "corpus_accessed": False,
}


receipt_path.write_text(
    json.dumps(
        receipt,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


# Manifest hashes the first four files.
manifest_entries = {}

for path in [
    narrative_path,
    trace_path,
    asset_path,
    receipt_path,
]:

    manifest_entries[
        str(
            path.relative_to(
                OUT
            )
        )
    ] = {
        "bytes":
            path.stat().st_size,

        "sha256":
            sha256_file(
                path
            ),
    }


manifest = {
    "schema":
        "stage26_p2_paper_facing_narrative_manifest_v1",

    "scientific_parent":
        EXPECTED_PARENT,

    "files":
        manifest_entries,
}


manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


print("Generated files:")

for path in [
    narrative_path,
    trace_path,
    asset_path,
    receipt_path,
    manifest_path,
]:

    print()
    print(
        path.name
    )

    print(
        "  bytes :",
        path.stat().st_size
    )

    print(
        "  SHA256:",
        sha256_file(path)
    )


# =============================================================================
# 9. FINAL SCIENTIFIC SAFETY AUDIT
# =============================================================================

banner("STAGE26-P2 :: FINAL SCIENTIFIC SAFETY AUDIT")

print("New timing/inference/memory : NONE")
print("Model loading                : NONE")
print("Corpus access                : NONE")
print("Complete E2E claim           : NONE")
print("Cross-group Pareto           : NONE")
print("Ratio confidence intervals   : NOT COMPUTED")
print("Missing-cost imputation      : NONE")
print("Post-hoc best batch          : NONE")
print("Narrative role               : DERIVED MANUSCRIPT SUPPORT")


# =============================================================================
# 10. STAGE EXACT FILE SET
# =============================================================================

banner("STAGE26-P2 :: STAGE EXACT PACKAGE")

expected_rel = sorted(
    str(
        p.relative_to(
            REPO
        )
    )
    for p in [
        narrative_path,
        trace_path,
        asset_path,
        receipt_path,
        manifest_path,
    ]
)


git(
    "add",
    "--",
    *expected_rel,
)


staged = sorted(
    x
    for x in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if x
)


print(
    "Staged files:",
    len(staged)
)

for x in staged:
    print(
        " ",
        x
    )


if staged != expected_rel:
    raise RuntimeError(
        "Unexpected staged file set."
    )


unstaged = git(
    "diff",
    "--name-only",
)

if unstaged:
    raise RuntimeError(
        "Unexpected unstaged tracked modifications:\n"
        + unstaged
    )


untracked = git(
    "ls-files",
    "--others",
    "--exclude-standard",
)

if untracked:
    raise RuntimeError(
        "Unexpected untracked files:\n"
        + untracked
    )


# =============================================================================
# 11. COMMIT
# =============================================================================

banner("STAGE26-P2 :: COMMIT")

git(
    "commit",
    "-m",
    COMMIT_MSG,
)


new_head = git(
    "rev-parse",
    "HEAD",
)

parent = git(
    "rev-parse",
    "HEAD^",
)

subject = git(
    "log",
    "-1",
    "--pretty=%s",
)


print("Parent :", parent)
print("HEAD   :", new_head)
print("Subject:", subject)


if parent != EXPECTED_PARENT:
    raise RuntimeError(
        "P2 commit parent mismatch."
    )

if subject != COMMIT_MSG:
    raise RuntimeError(
        "P2 commit subject mismatch."
    )


# =============================================================================
# 12. PUSH SECURELY
# =============================================================================

banner("STAGE26-P2 :: PUSH")

from kaggle_secrets import UserSecretsClient


token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)

if not token:
    raise RuntimeError(
        "GITHUB_TOKEN unavailable."
    )


fd, askpass_name = tempfile.mkstemp(
    prefix="stage26_p2_askpass_",
    suffix=".sh",
)

os.close(fd)

askpass = Path(
    askpass_name
)


try:

    askpass.write_text(
        "#!/bin/sh\n"
        "case \"$1\" in\n"
        "  *Username*) printf '%s\\n' 'x-access-token' ;;\n"
        "  *Password*) printf '%s\\n' \"$STAGE26_GITHUB_TOKEN\" ;;\n"
        "  *) printf '%s\\n' '' ;;\n"
        "esac\n",
        encoding="utf-8",
    )

    askpass.chmod(
        askpass.stat().st_mode
        | stat.S_IXUSR
    )


    env = os.environ.copy()

    env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    env[
        "STAGE26_GITHUB_TOKEN"
    ] = token


    push_output = git(
        "push",
        "origin",
        "main",
        env=env,
    )

    print(
        push_output
    )


finally:

    askpass.unlink(
        missing_ok=True
    )

    token = None


# =============================================================================
# 13. REMOTE BYTE VERIFICATION
# =============================================================================

banner("STAGE26-P2 :: REMOTE BYTE VERIFICATION")

git(
    "fetch",
    "origin",
    "main",
)


local_head = git(
    "rev-parse",
    "HEAD",
)

remote_head = git(
    "rev-parse",
    "origin/main",
)


print("Local HEAD :", local_head)
print("origin/main:", remote_head)


if local_head != remote_head:
    raise RuntimeError(
        "HEAD != origin/main after push."
    )


for path in [
    narrative_path,
    trace_path,
    asset_path,
    receipt_path,
    manifest_path,
]:

    rel = str(
        path.relative_to(
            REPO
        )
    )

    p = subprocess.run(
        [
            "git",
            "show",
            f"origin/main:{rel}",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            f"Remote read failed: {rel}\n"
            + p.stderr.decode(
                "utf-8",
                errors="replace",
            )
        )


    remote_sha = hashlib.sha256(
        p.stdout
    ).hexdigest()

    local_sha = sha256_file(
        path
    )


    print(
        f"{'PASS' if local_sha == remote_sha else 'FAIL'} "
        f"{rel}"
    )

    print(
        "  local :",
        local_sha
    )

    print(
        "  remote:",
        remote_sha
    )


    if local_sha != remote_sha:
        raise RuntimeError(
            f"Remote byte mismatch: {rel}"
        )


# =============================================================================
# 14. FINAL STATE
# =============================================================================

banner("STAGE26-P2 COMPLETE :: PAPER-FACING NARRATIVE FROZEN")

final_status = git(
    "status",
    "--porcelain",
)


print(
    "P2 narrative anchor        :",
    new_head,
)

print(
    "Scientific parent          :",
    EXPECTED_PARENT,
)

print(
    "HEAD == origin/main        :",
    local_head == remote_head == new_head,
)

print(
    "Repo clean                 :",
    final_status == "",
)

print()
print(
    "Stage26 measurement science: FROZEN"
)

print(
    "Paper-facing narrative     : FROZEN"
)

print(
    "New science introduced     : NO"
)

print()
print(
    "NEXT PHASE:"
)

print(
    "  INTEGRATE THE COMPLETE PAPER:"
)

print(
    "  Abstract -> Introduction -> Related Work -> "
    "Methodology -> Validation-Safe Ablations -> "
    "Results -> Explainability -> Deployment -> "
    "Limitations -> Conclusion"
)


if final_status:
    raise RuntimeError(
        "Repository not clean after P2."
    )


STAGE26-P2 :: DURABLE PARENT GATE
Expected parent : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
Local HEAD      : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
origin/main     : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
Repo clean      : True

STAGE26-P2 :: LOAD FROZEN PUBLICATION SOURCES
gpu_warm     results/stage26_deployment_profiling/stage26_g2_final_gpu_publication_closure/T26_GPU_WARM_INFERENCE_MEMORY.csv
             fd16546ba1b2b4130fc2fdff7095302e38a015770feed1965a4d7df8e4ff8597
matched      results/stage26_deployment_profiling/stage26_g2_final_gpu_publication_closure/T26_CPU1_GPU_MATCHED_COMPARISON.csv
             24cbd9a22ad58b85782485a0c08109a7606429b8fc09674056ab7c9cf29edc3b
gpu_status   results/stage26_deployment_profiling/stage26_g2_final_gpu_publication_closure/T26_GPU_BACKEND_RESOURCE_STATUS.csv
             3f6922821b4d3f01ff8b84405584acea0d8270a5dba7f4402aaa7ee1fafc9ade
batch1       results/stage26_deployment_profiling/stage26_g2_final_gpu_publication_closure/T26_CPU1_GPU_B

In [7]:
# =============================================================================
# STAGE26-P1P :: PUSH EXISTING P1 MANUSCRIPT-SUPPORT FILES UNCHANGED
#
# IMPORTANT
#   - Stage26 science is ALREADY COMPLETE.
#   - This does NOT create another scientific Stage26 stage.
#   - Copies the existing P1 files BYTE-FOR-BYTE without rewriting them.
#   - No inference / timing / corpus / model loading.
# =============================================================================

from __future__ import annotations

import hashlib
import os
import shutil
import stat
import subprocess
import tempfile
from pathlib import Path


REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")

STAGE26_FINAL_SCIENTIFIC_ANCHOR = (
    "9e8354ecc9cfa72c28aa037e5d2053de422bf7a2"
)

SRC = Path(
    "/kaggle/working/stage26_manuscript_draft"
)

DST = (
    REPO
    / "results"
    / "manuscript_support"
    / "stage26_deployment_draft"
)

COMMIT_MSG = (
    "docs: preserve Stage26 deployment manuscript draft"
)

EXPECTED = {
    "stage26_results_deployment_discussion_draft.md":
        "eb4c5776fb5de0579c325c530e4bfc21ef8ff82057ceea2bf32b9496de96418a",

    "stage26_claim_traceability.csv":
        "f675537ae869493d9a7aa914ac7bdf223e107fd9db150e222599a0b1652291c4",

    "stage26_manuscript_asset_plan.csv":
        "f6a207df19b85b0918275cc613f71aad23e7dde1152e0bd77983c7a7fff48509",
}


def banner(text):
    print()
    print("=" * 120)
    print(text)
    print("=" * 120)


def run(cmd, *, cwd=None, env=None, check=True):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(map(str, cmd))}\n{p.stdout}"
        )

    return p.stdout.strip()


def git(*args, env=None, check=True):
    return run(
        ["git", *args],
        cwd=REPO,
        env=env,
        check=check,
    )


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for block in iter(
            lambda: f.read(16 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


# =============================================================================
# 1. EXACT STAGE26 FINAL PARENT GATE
# =============================================================================

banner("P1P :: REPOSITORY GATE")

head = git("rev-parse", "HEAD")
origin = git("rev-parse", "origin/main")
status = git("status", "--porcelain")

print("Stage26 scientific anchor :", STAGE26_FINAL_SCIENTIFIC_ANCHOR)
print("Local HEAD                :", head)
print("origin/main               :", origin)
print("Repo clean                :", status == "")

if head != STAGE26_FINAL_SCIENTIFIC_ANCHOR:
    raise RuntimeError(
        "HEAD is no longer the final Stage26 scientific anchor."
    )

if origin != STAGE26_FINAL_SCIENTIFIC_ANCHOR:
    raise RuntimeError(
        "origin/main is no longer the final Stage26 scientific anchor."
    )

if status:
    raise RuntimeError(
        "Repository is dirty."
    )


# =============================================================================
# 2. VERIFY ORIGINAL P1 FILES BEFORE COPYING
# =============================================================================

banner("P1P :: VERIFY EXISTING P1 FILES")

for name, expected_sha in EXPECTED.items():

    path = SRC / name

    if not path.is_file():
        raise FileNotFoundError(path)

    actual_sha = sha256_file(path)

    print(name)
    print("  expected:", expected_sha)
    print("  actual  :", actual_sha)
    print("  PASS    :", actual_sha == expected_sha)

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"P1 source file changed: {name}"
        )


# =============================================================================
# 3. COPY BYTE-FOR-BYTE — NO CONTENT TRANSFORMATION
# =============================================================================

banner("P1P :: PRESERVE P1 FILES BYTE-FOR-BYTE")

if DST.exists():
    raise RuntimeError(
        f"Destination already exists: {DST}"
    )

DST.mkdir(
    parents=True,
    exist_ok=False,
)


for name, expected_sha in EXPECTED.items():

    src = SRC / name
    dst = DST / name

    shutil.copyfile(
        src,
        dst,
    )

    src_sha = sha256_file(src)
    dst_sha = sha256_file(dst)

    print(name)
    print("  source     :", src_sha)
    print("  repository :", dst_sha)
    print("  identical  :", src_sha == dst_sha == expected_sha)

    if not (
        src_sha
        ==
        dst_sha
        ==
        expected_sha
    ):
        raise RuntimeError(
            f"Byte identity failure: {name}"
        )


# =============================================================================
# 4. STAGE EXACTLY THESE THREE FILES
# =============================================================================

banner("P1P :: STAGE EXACT FILE SET")

relpaths = sorted(
    str(
        (DST / name).relative_to(REPO)
    )
    for name in EXPECTED
)

git(
    "add",
    "--",
    *relpaths,
)


staged = sorted(
    x
    for x in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if x
)


print("Expected staged:", len(relpaths))
print("Actual staged  :", len(staged))

for x in staged:
    print(" ", x)


if staged != relpaths:
    raise RuntimeError(
        "Unexpected staged file set."
    )


if git("diff", "--name-only"):
    raise RuntimeError(
        "Unexpected unstaged tracked changes."
    )


untracked = git(
    "ls-files",
    "--others",
    "--exclude-standard",
)

if untracked:
    raise RuntimeError(
        "Unexpected untracked files:\n" + untracked
    )


# =============================================================================
# 5. COMMIT
# =============================================================================

banner("P1P :: COMMIT")

git(
    "commit",
    "-m",
    COMMIT_MSG,
)

new_head = git(
    "rev-parse",
    "HEAD",
)

parent = git(
    "rev-parse",
    "HEAD^",
)


print("Scientific Stage26 anchor :", STAGE26_FINAL_SCIENTIFIC_ANCHOR)
print("Commit parent             :", parent)
print("New repository HEAD       :", new_head)


if parent != STAGE26_FINAL_SCIENTIFIC_ANCHOR:
    raise RuntimeError(
        "Unexpected commit parent."
    )


# =============================================================================
# 6. PUSH SECURELY
# =============================================================================

banner("P1P :: PUSH")

from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)

if not token:
    raise RuntimeError(
        "GITHUB_TOKEN unavailable."
    )


fd, askpass_name = tempfile.mkstemp(
    prefix="stage26_p1p_",
    suffix=".sh",
)

os.close(fd)

askpass = Path(askpass_name)


try:

    askpass.write_text(
        "#!/bin/sh\n"
        "case \"$1\" in\n"
        "  *Username*) printf '%s\\n' 'x-access-token' ;;\n"
        "  *Password*) printf '%s\\n' \"$GITHUB_TOKEN_VALUE\" ;;\n"
        "  *) printf '%s\\n' '' ;;\n"
        "esac\n",
        encoding="utf-8",
    )

    askpass.chmod(
        askpass.stat().st_mode
        | stat.S_IXUSR
    )

    env = os.environ.copy()

    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["GITHUB_TOKEN_VALUE"] = token

    print(
        git(
            "push",
            "origin",
            "main",
            env=env,
        )
    )

finally:

    askpass.unlink(
        missing_ok=True
    )

    token = None


# =============================================================================
# 7. REMOTE BYTE VERIFICATION
# =============================================================================

banner("P1P :: REMOTE BYTE VERIFICATION")

git(
    "fetch",
    "origin",
    "main",
)

local_head = git(
    "rev-parse",
    "HEAD",
)

remote_head = git(
    "rev-parse",
    "origin/main",
)


print("Local HEAD :", local_head)
print("origin/main:", remote_head)


if local_head != remote_head:
    raise RuntimeError(
        "Local HEAD != origin/main."
    )


for name, expected_sha in EXPECTED.items():

    rel = str(
        (DST / name).relative_to(REPO)
    )

    p = subprocess.run(
        [
            "git",
            "show",
            f"origin/main:{rel}",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            f"Remote read failed: {rel}"
        )

    remote_sha = hashlib.sha256(
        p.stdout
    ).hexdigest()

    local_sha = sha256_file(
        DST / name
    )

    print(name)
    print("  expected:", expected_sha)
    print("  local   :", local_sha)
    print("  remote  :", remote_sha)
    print(
        "  PASS    :",
        expected_sha == local_sha == remote_sha,
    )

    if not (
        expected_sha
        ==
        local_sha
        ==
        remote_sha
    ):
        raise RuntimeError(
            f"Remote byte mismatch: {name}"
        )


# =============================================================================
# 8. FINAL STATE
# =============================================================================

banner("P1P COMPLETE :: STAGE26 LEFT AS-IS")

final_status = git(
    "status",
    "--porcelain",
)

print("STAGE26 SCIENTIFIC ANCHOR :", STAGE26_FINAL_SCIENTIFIC_ANCHOR)
print("NEW REPOSITORY HEAD       :", new_head)
print("HEAD == origin/main       :", local_head == remote_head == new_head)
print("Repo clean                :", final_status == "")

print()
print("P1 files preserved        : 3 / 3")
print("P1 contents modified      : NO")
print("New Stage26 science       : NO")
print("Stage26 reopened          : NO")
print()

print("STAGE26                   : COMPLETE / CLOSED")
print("NEXT                      : BEGIN NEXT SCIENTIFIC STAGE")


if final_status:
    raise RuntimeError(
        "Repository not clean after push."
    )


P1P :: REPOSITORY GATE
Stage26 scientific anchor : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
Local HEAD                : e31f9c668df69bbd243165d071b431a6b5c54eba
origin/main               : e31f9c668df69bbd243165d071b431a6b5c54eba
Repo clean                : True


RuntimeError: HEAD is no longer the final Stage26 scientific anchor.

In [8]:
# =============================================================================
# STAGE26-P1P-R1 :: PRESERVE ORIGINAL P1 FILES ON CURRENT REPOSITORY HEAD
#
# Current history is preserved.
# NO reset / revert / force-push.
#
# Scientific Stage26 anchor:
#   9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
#
# Existing derived documentation commit:
#   e31f9c668df69bbd243165d071b431a6b5c54eba
#
# This cell:
#   - verifies both identities
#   - verifies original P1 temporary files byte-for-byte
#   - copies them unchanged into manuscript_support/
#   - commits them on top of current HEAD
#   - pushes
#   - remotely byte-verifies
#
# NO SCIENCE
# NO INFERENCE
# NO TIMING
# NO CORPUS
# =============================================================================

from __future__ import annotations

import hashlib
import os
import shutil
import stat
import subprocess
import tempfile
from pathlib import Path


REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SCIENTIFIC_STAGE26_ANCHOR = (
    "9e8354ecc9cfa72c28aa037e5d2053de422bf7a2"
)

EXPECTED_CURRENT_HEAD = (
    "e31f9c668df69bbd243165d071b431a6b5c54eba"
)

SRC = Path(
    "/kaggle/working/stage26_manuscript_draft"
)

DST = (
    REPO
    / "results"
    / "manuscript_support"
    / "stage26_deployment_draft"
)

COMMIT_MSG = (
    "docs: preserve original Stage26 deployment draft"
)

EXPECTED = {
    "stage26_results_deployment_discussion_draft.md":
        "eb4c5776fb5de0579c325c530e4bfc21ef8ff82057ceea2bf32b9496de96418a",

    "stage26_claim_traceability.csv":
        "f675537ae869493d9a7aa914ac7bdf223e107fd9db150e222599a0b1652291c4",

    "stage26_manuscript_asset_plan.csv":
        "f6a207df19b85b0918275cc613f71aad23e7dde1152e0bd77983c7a7fff48509",
}


def banner(text):
    print()
    print("=" * 120)
    print(text)
    print("=" * 120)


def run(cmd, *, cwd=None, env=None, check=True):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(map(str, cmd))}\n\n{p.stdout}"
        )

    return p.stdout.strip()


def git(*args, env=None, check=True):
    return run(
        ["git", *args],
        cwd=REPO,
        env=env,
        check=check,
    )


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for block in iter(
            lambda: f.read(16 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


# =============================================================================
# 1. CURRENT HISTORY GATE
# =============================================================================

banner("P1P-R1 :: CURRENT HISTORY GATE")

git(
    "fetch",
    "origin",
    "main",
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)

parent = git(
    "rev-parse",
    f"{EXPECTED_CURRENT_HEAD}^",
)

subject = git(
    "show",
    "-s",
    "--format=%s",
    EXPECTED_CURRENT_HEAD,
)


print("Scientific Stage26 anchor :", SCIENTIFIC_STAGE26_ANCHOR)
print("Expected current HEAD     :", EXPECTED_CURRENT_HEAD)
print("Local HEAD                :", head)
print("origin/main               :", origin)
print("Current HEAD parent       :", parent)
print("Current HEAD subject      :", subject)
print("Repo clean                :", status == "")


if head != EXPECTED_CURRENT_HEAD:
    raise RuntimeError(
        "Local HEAD differs from the already-pushed documentation commit."
    )

if origin != EXPECTED_CURRENT_HEAD:
    raise RuntimeError(
        "origin/main differs from the already-pushed documentation commit."
    )

if parent != SCIENTIFIC_STAGE26_ANCHOR:
    raise RuntimeError(
        "Current documentation commit does not directly descend "
        "from the frozen Stage26 scientific anchor."
    )

if status:
    raise RuntimeError(
        "Repository is dirty before P1 preservation."
    )


print()
print("Scientific Stage26 anchor preserved : YES")
print("Existing documentation history kept : YES")
print("Reset/revert required                : NO")


# =============================================================================
# 2. VERIFY ORIGINAL P1 TEMP FILES
# =============================================================================

banner("P1P-R1 :: VERIFY ORIGINAL P1 FILES")

for name, expected_sha in EXPECTED.items():

    path = SRC / name

    if not path.is_file():
        raise FileNotFoundError(
            path
        )

    actual_sha = sha256_file(
        path
    )

    print(name)
    print("  expected:", expected_sha)
    print("  actual  :", actual_sha)
    print("  PASS    :", actual_sha == expected_sha)

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"Original P1 file changed: {name}"
        )


# =============================================================================
# 3. COPY ORIGINAL P1 FILES BYTE-FOR-BYTE
# =============================================================================

banner("P1P-R1 :: COPY P1 FILES BYTE-FOR-BYTE")

if DST.exists():
    raise RuntimeError(
        f"Destination already exists: {DST}"
    )

DST.mkdir(
    parents=True,
    exist_ok=False,
)


for name, expected_sha in EXPECTED.items():

    src = SRC / name
    dst = DST / name

    shutil.copyfile(
        src,
        dst,
    )

    src_sha = sha256_file(
        src
    )

    dst_sha = sha256_file(
        dst
    )

    print(name)
    print("  source SHA :", src_sha)
    print("  copied SHA :", dst_sha)
    print(
        "  identical  :",
        src_sha == dst_sha == expected_sha,
    )

    if not (
        src_sha
        ==
        dst_sha
        ==
        expected_sha
    ):
        raise RuntimeError(
            f"Byte identity failed: {name}"
        )


# =============================================================================
# 4. STAGE EXACTLY THREE FILES
# =============================================================================

banner("P1P-R1 :: STAGE EXACT ORIGINAL P1 PACKAGE")

relpaths = sorted(
    str(
        (DST / name).relative_to(
            REPO
        )
    )
    for name in EXPECTED
)


git(
    "add",
    "--",
    *relpaths,
)


staged = sorted(
    x
    for x in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if x
)


print("Expected staged files :", len(relpaths))
print("Actual staged files   :", len(staged))

for x in staged:
    print(" ", x)


if staged != relpaths:
    raise RuntimeError(
        "Unexpected staged file set."
    )


unstaged = git(
    "diff",
    "--name-only",
)

if unstaged:
    raise RuntimeError(
        "Unexpected unstaged tracked modifications:\n"
        + unstaged
    )


untracked = git(
    "ls-files",
    "--others",
    "--exclude-standard",
)

if untracked:
    raise RuntimeError(
        "Unexpected untracked files:\n"
        + untracked
    )


# =============================================================================
# 5. COMMIT
# =============================================================================

banner("P1P-R1 :: COMMIT")

git(
    "commit",
    "-m",
    COMMIT_MSG,
)


new_head = git(
    "rev-parse",
    "HEAD",
)

new_parent = git(
    "rev-parse",
    "HEAD^",
)

new_subject = git(
    "show",
    "-s",
    "--format=%s",
    new_head,
)


print("Scientific anchor :", SCIENTIFIC_STAGE26_ANCHOR)
print("Previous HEAD     :", EXPECTED_CURRENT_HEAD)
print("Commit parent     :", new_parent)
print("New HEAD          :", new_head)
print("Subject           :", new_subject)


if new_parent != EXPECTED_CURRENT_HEAD:
    raise RuntimeError(
        "P1 preservation commit has unexpected parent."
    )

if new_subject != COMMIT_MSG:
    raise RuntimeError(
        "Unexpected commit subject."
    )


# =============================================================================
# 6. PUSH
# =============================================================================

banner("P1P-R1 :: PUSH")

from kaggle_secrets import UserSecretsClient


token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)

if not token:
    raise RuntimeError(
        "GITHUB_TOKEN unavailable."
    )


fd, askpass_name = tempfile.mkstemp(
    prefix="stage26_p1p_r1_",
    suffix=".sh",
)

os.close(fd)

askpass = Path(
    askpass_name
)


try:

    askpass.write_text(
        "#!/bin/sh\n"
        "case \"$1\" in\n"
        "  *Username*) printf '%s\\n' 'x-access-token' ;;\n"
        "  *Password*) printf '%s\\n' \"$TOKEN_VALUE\" ;;\n"
        "  *) printf '%s\\n' '' ;;\n"
        "esac\n",
        encoding="utf-8",
    )

    askpass.chmod(
        askpass.stat().st_mode
        | stat.S_IXUSR
    )


    env = os.environ.copy()

    env["GIT_ASKPASS"] = str(
        askpass
    )

    env["GIT_TERMINAL_PROMPT"] = "0"

    env["TOKEN_VALUE"] = token


    print(
        git(
            "push",
            "origin",
            "main",
            env=env,
        )
    )


finally:

    askpass.unlink(
        missing_ok=True
    )

    token = None


# =============================================================================
# 7. REMOTE BYTE VERIFICATION
# =============================================================================

banner("P1P-R1 :: REMOTE BYTE VERIFICATION")

git(
    "fetch",
    "origin",
    "main",
)


local_head = git(
    "rev-parse",
    "HEAD",
)

remote_head = git(
    "rev-parse",
    "origin/main",
)


print("Local HEAD :", local_head)
print("origin/main:", remote_head)


if not (
    local_head
    ==
    remote_head
    ==
    new_head
):
    raise RuntimeError(
        "Push did not establish expected HEAD remotely."
    )


for name, expected_sha in EXPECTED.items():

    rel = str(
        (DST / name).relative_to(
            REPO
        )
    )

    p = subprocess.run(
        [
            "git",
            "show",
            f"origin/main:{rel}",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            f"Unable to read remote file: {rel}"
        )


    remote_sha = hashlib.sha256(
        p.stdout
    ).hexdigest()

    local_sha = sha256_file(
        DST / name
    )


    print(name)
    print("  expected:", expected_sha)
    print("  local   :", local_sha)
    print("  remote  :", remote_sha)
    print(
        "  PASS    :",
        expected_sha
        ==
        local_sha
        ==
        remote_sha,
    )


    if not (
        expected_sha
        ==
        local_sha
        ==
        remote_sha
    ):
        raise RuntimeError(
            f"Remote byte verification failed: {name}"
        )


# =============================================================================
# 8. FINAL STATE
# =============================================================================

banner("P1P-R1 COMPLETE :: ORIGINAL P1 PRESERVED")

final_status = git(
    "status",
    "--porcelain",
)


print("STAGE26 SCIENTIFIC ANCHOR :", SCIENTIFIC_STAGE26_ANCHOR)
print("Prior documentation HEAD  :", EXPECTED_CURRENT_HEAD)
print("New repository HEAD       :", new_head)

print()
print(
    "HEAD == origin/main       :",
    local_head
    ==
    remote_head
    ==
    new_head,
)

print(
    "Repo clean                :",
    final_status == "",
)

print()
print("Original P1 files         : 3 / 3")
print("Original P1 bytes changed : NO")
print("Existing P2 commit removed: NO")
print("Stage26 science changed   : NO")
print("New experiment            : NO")

print()
print("STAGE26                   : COMPLETE / CLOSED")
print("NEXT                      : NEXT SCIENTIFIC STAGE")


if final_status:
    raise RuntimeError(
        "Repository not clean after P1 preservation."
    )


P1P-R1 :: CURRENT HISTORY GATE
Scientific Stage26 anchor : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
Expected current HEAD     : e31f9c668df69bbd243165d071b431a6b5c54eba
Local HEAD                : e31f9c668df69bbd243165d071b431a6b5c54eba
origin/main               : e31f9c668df69bbd243165d071b431a6b5c54eba
Current HEAD parent       : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
Current HEAD subject      : stage26: freeze paper-facing deployment narrative
Repo clean                : True

Scientific Stage26 anchor preserved : YES
Existing documentation history kept : YES
Reset/revert required                : NO

P1P-R1 :: VERIFY ORIGINAL P1 FILES
stage26_results_deployment_discussion_draft.md
  expected: eb4c5776fb5de0579c325c530e4bfc21ef8ff82057ceea2bf32b9496de96418a
  actual  : eb4c5776fb5de0579c325c530e4bfc21ef8ff82057ceea2bf32b9496de96418a
  PASS    : True
stage26_claim_traceability.csv
  expected: f675537ae869493d9a7aa914ac7bdf223e107fd9db150e222599a0b1652291c4
  actual  : f675537ae86

In [9]:
# =============================================================================
# STAGE26-PC0 :: CANONICAL PUBLICATION CLOSEOUT BUILD
#
# POST-CLOSURE ARCHIVAL / PUBLICATION PACKAGING ONLY
#
# Creates the previously missing canonical structure:
#
#   results/stage26_deployment_profiling/
#       stage26_8_gpu_protocol_lock/
#       stage26_9_gpu_inference/
#       stage26_10_gpu_memory/
#       stage26_11_gpu_cpu_comparison/
#       stage26_12_final_synthesis/
#       stage26_publication_package/
#
#   docs/
#       STAGE26_MANUSCRIPT_INTEGRATION.md
#       STAGE26_PUBLICATION_CLOSEOUT.md
#
#   figures/
#       stage26_deployment_profiling/
#
# IMPORTANT:
#   stage26_8_gpu_protocol_lock is explicitly marked as a POST-CLOSURE
#   ARCHIVAL RECONSTRUCTION / INDEX. It is NOT represented as a prospective
#   preregistration that existed before GPU measurement.
#
# NO GPU
# NO inference
# NO timing
# NO model loading
# NO corpus access
# NO new scientific measurement
# NO Git commit/push in this cell
# =============================================================================

from __future__ import annotations

import csv
import hashlib
import json
import os
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path


# =============================================================================
# 0. FROZEN IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_CURRENT_HEAD = (
    "3bf4c6e9e9cb2e6ccfeb108e35cbf5fbeded6fd4"
)

STAGE26_SCIENTIFIC_ANCHOR = (
    "9e8354ecc9cfa72c28aa037e5d2053de422bf7a2"
)

STAGE26_CPU_CLOSURE_ANCHOR = (
    "304e5613627a744cbd5d369857f8ac5667a520eb"
)

STAGE26_GPU_G1_ANCHOR = (
    "79b9c23c91ef7185d35b700e6f6f84aefb54ab35"
)


BASE = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
)

CPU_TABLES_SRC = (
    BASE
    / "stage26_8d3_cpu_publication_tables"
)

CPU_FIGURES_SRC = (
    BASE
    / "stage26_8d4_cpu_publication_figures"
)

G1_SRC = (
    BASE
    / "stage26_g1_gpu_warm_profile"
)

G2_SRC = (
    BASE
    / "stage26_g2_final_gpu_publication_closure"
)

P2_SRC = (
    BASE
    / "stage26_p2_paper_facing_narrative"
)

P1_SRC = (
    REPO
    / "results"
    / "manuscript_support"
    / "stage26_deployment_draft"
)


# Temporary build is OUTSIDE repository.
BUILD = Path(
    "/kaggle/working/stage26_pc0_closeout_build"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def banner(text):
    print()
    print("=" * 124)
    print(text)
    print("=" * 124)


def run(cmd, *, cwd=REPO, check=True):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(map(str, cmd))}\n\n"
            f"{p.stdout}"
        )

    return p.stdout.strip()


def git(*args, check=True):
    return run(
        ["git", *args],
        check=check,
    )


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for block in iter(
            lambda: f.read(16 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def human_bytes(n):
    n = int(n)

    if n >= 1024**3:
        return f"{n / 1024**3:.3f} GiB"

    if n >= 1024**2:
        return f"{n / 1024**2:.3f} MiB"

    if n >= 1024:
        return f"{n / 1024:.3f} KiB"

    return f"{n} B"


def ensure_file(path):
    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(path)

    return path


def ensure_dir(path):
    path = Path(path)

    if not path.is_dir():
        raise FileNotFoundError(path)

    return path


def relative_repo(path):
    return str(
        Path(path).relative_to(REPO)
    )


def copy_verified(src, dst, role, provenance):
    src = ensure_file(src)
    dst = Path(dst)

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    src_sha = sha256_file(src)

    shutil.copyfile(
        src,
        dst,
    )

    dst_sha = sha256_file(dst)

    if src_sha != dst_sha:
        raise RuntimeError(
            f"Byte-copy mismatch:\n"
            f"source={src}\n"
            f"dest={dst}"
        )

    provenance.append(
        {
            "role": role,
            "source_repository_path":
                relative_repo(src),

            "destination_repository_path":
                str(
                    dst.relative_to(
                        BUILD
                    )
                ),

            "bytes":
                int(
                    dst.stat().st_size
                ),

            "sha256":
                dst_sha,

            "copy_mode":
                "BYTE_FOR_BYTE",
        }
    )


def write_text_file(path, text):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_text(
        text.rstrip() + "\n",
        encoding="utf-8",
    )


# =============================================================================
# 2. REPOSITORY / HISTORY GATE
# =============================================================================

banner("STAGE26-PC0 :: REPOSITORY / HISTORY GATE")

git(
    "fetch",
    "origin",
    "main",
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print("Expected current HEAD    :", EXPECTED_CURRENT_HEAD)
print("Local HEAD               :", head)
print("origin/main              :", origin)
print("Repository clean         :", status == "")

print()
print("Stage26 scientific anchor:", STAGE26_SCIENTIFIC_ANCHOR)
print("Stage26 CPU anchor       :", STAGE26_CPU_CLOSURE_ANCHOR)
print("Stage26 G1 GPU anchor    :", STAGE26_GPU_G1_ANCHOR)


if head != EXPECTED_CURRENT_HEAD:
    raise RuntimeError(
        "Unexpected repository HEAD."
    )

if origin != EXPECTED_CURRENT_HEAD:
    raise RuntimeError(
        "origin/main differs from expected HEAD."
    )

if status:
    raise RuntimeError(
        "Repository must be clean before closeout packaging."
    )


for anchor, label in [
    (
        STAGE26_SCIENTIFIC_ANCHOR,
        "Stage26 scientific anchor",
    ),
    (
        STAGE26_CPU_CLOSURE_ANCHOR,
        "Stage26 CPU closure anchor",
    ),
    (
        STAGE26_GPU_G1_ANCHOR,
        "Stage26 G1 GPU anchor",
    ),
]:

    rc = subprocess.run(
        [
            "git",
            "merge-base",
            "--is-ancestor",
            anchor,
            head,
        ],
        cwd=REPO,
        check=False,
    ).returncode

    print(
        f"{label:<30s}:",
        rc == 0,
    )

    if rc != 0:
        raise RuntimeError(
            f"{label} is not an ancestor of current HEAD."
        )


# =============================================================================
# 3. VERIFY FROZEN SOURCE DIRECTORIES
# =============================================================================

banner("STAGE26-PC0 :: VERIFY FROZEN SOURCE DIRECTORIES")

for path in [
    CPU_TABLES_SRC,
    CPU_FIGURES_SRC,
    G1_SRC,
    G2_SRC,
    P2_SRC,
    P1_SRC,
]:
    ensure_dir(path)
    print(
        "PASS",
        relative_repo(path),
    )


# Confirm none of the scientific source directories changed after the
# Stage26 scientific closure commit.
scientific_paths = [
    relative_repo(
        CPU_TABLES_SRC
    ),
    relative_repo(
        CPU_FIGURES_SRC
    ),
    relative_repo(
        G1_SRC
    ),
    relative_repo(
        G2_SRC
    ),
]


p = subprocess.run(
    [
        "git",
        "diff",
        "--quiet",
        STAGE26_SCIENTIFIC_ANCHOR,
        "HEAD",
        "--",
        *scientific_paths,
    ],
    cwd=REPO,
    check=False,
)


print()
print(
    "Frozen scientific source directories unchanged "
    "since Stage26 closure:",
    p.returncode == 0,
)


if p.returncode != 0:
    print(
        git(
            "diff",
            "--name-status",
            STAGE26_SCIENTIFIC_ANCHOR,
            "HEAD",
            "--",
            *scientific_paths,
        )
    )

    raise RuntimeError(
        "Stage26 frozen scientific sources changed after closure."
    )


# =============================================================================
# 4. VERIFY EXPECTED SOURCE GEOMETRY
# =============================================================================

banner("STAGE26-PC0 :: SOURCE GEOMETRY AUDIT")


cpu_tables = sorted(
    CPU_TABLES_SRC.glob(
        "T26_*.csv"
    )
)

cpu_pngs = sorted(
    CPU_FIGURES_SRC.glob(
        "F26_*.png"
    )
)

cpu_pdfs = sorted(
    CPU_FIGURES_SRC.glob(
        "F26_*.pdf"
    )
)


print(
    "CPU publication tables:",
    len(cpu_tables),
)

print(
    "CPU PNG figures       :",
    len(cpu_pngs),
)

print(
    "CPU PDF figures       :",
    len(cpu_pdfs),
)


if len(cpu_tables) != 8:
    raise RuntimeError(
        f"Expected 8 CPU publication tables, got {len(cpu_tables)}"
    )

if len(cpu_pngs) != 8:
    raise RuntimeError(
        f"Expected 8 CPU PNG figures, got {len(cpu_pngs)}"
    )

if len(cpu_pdfs) != 8:
    raise RuntimeError(
        f"Expected 8 CPU PDF figures, got {len(cpu_pdfs)}"
    )


G1_REQUIRED = [
    "stage26_g1_gpu_condition_summary.csv",
    "stage26_g1_gpu_environment.json",
    "stage26_g1_gpu_execution_plan.csv",
    "stage26_g1_gpu_profile_manifest.json",
    "stage26_g1_gpu_profile_receipt.json",
    "stage26_g1_gpu_raw_timing.csv",
    "stage26_g1_gpu_worker_results.json",
]


G2_REQUIRED = [
    "T26_GPU_WARM_INFERENCE_MEMORY.csv",
    "T26_CPU1_GPU_MATCHED_COMPARISON.csv",
    "T26_GPU_BACKEND_RESOURCE_STATUS.csv",
    "T26_CPU1_GPU_BATCH1_ANCHOR.csv",

    "F26_CPU1_GPU_P95_SPEEDUP.png",
    "F26_CPU1_GPU_P95_SPEEDUP.pdf",
    "F26_GPU_DELTA_PEAK_MEMORY.png",
    "F26_GPU_DELTA_PEAK_MEMORY.pdf",

    "stage26_g2_final_gpu_publication_index.json",
    "stage26_g2_final_stage26_closure_receipt.json",
    "stage26_g2_final_gpu_publication_closure_manifest.json",
]


for name in G1_REQUIRED:
    ensure_file(
        G1_SRC / name
    )


for name in G2_REQUIRED:
    ensure_file(
        G2_SRC / name
    )


print(
    "G1 required files      :",
    len(G1_REQUIRED),
    "/ 7"
)

print(
    "G2 required files      :",
    len(G2_REQUIRED),
    "/ 11"
)


# =============================================================================
# 5. VERIFY KNOWN FROZEN G1/G2 SHA256 VALUES
# =============================================================================

banner("STAGE26-PC0 :: FROZEN G1/G2 HASH AUDIT")


EXPECTED_SHA256 = {
    # -------------------------------------------------------------------------
    # G1 GPU profile
    # -------------------------------------------------------------------------
    G1_SRC / "stage26_g1_gpu_execution_plan.csv":
        "3d19e36b9b6000de853e291468a46b668df4301dac4c9304a3b72dda04469273",

    G1_SRC / "stage26_g1_gpu_raw_timing.csv":
        "c41ffa687b3b779762f0adae2808d5e6489c4402d57fd8029d1c45f601e5565c",

    G1_SRC / "stage26_g1_gpu_condition_summary.csv":
        "b355ce66882d1ebcc0f4a1a3cd030b81f2dac25f462761d22bcd0428b3bc2625",

    G1_SRC / "stage26_g1_gpu_worker_results.json":
        "4253b8a9992f413e8c77225f54fd7ace7078b19e6fc459ff7211251b09ce98ac",

    G1_SRC / "stage26_g1_gpu_environment.json":
        "dde88a5ebafe82ccca466db967a4797d440bcbfa006429427d64ecce6c7b6e5f",

    G1_SRC / "stage26_g1_gpu_profile_receipt.json":
        "4ed7c3999ef6f6fffd8361ebca04f9a4398565a00953c50d97acae922c8502c2",

    G1_SRC / "stage26_g1_gpu_profile_manifest.json":
        "7aa8ac38b9cdc38aad43d4380f850c334f5235cac61378289c3a8295457e6048",

    # -------------------------------------------------------------------------
    # G2 final closure
    # -------------------------------------------------------------------------
    G2_SRC / "T26_GPU_WARM_INFERENCE_MEMORY.csv":
        "fd16546ba1b2b4130fc2fdff7095302e38a015770feed1965a4d7df8e4ff8597",

    G2_SRC / "T26_CPU1_GPU_MATCHED_COMPARISON.csv":
        "24cbd9a22ad58b85782485a0c08109a7606429b8fc09674056ab7c9cf29edc3b",

    G2_SRC / "T26_GPU_BACKEND_RESOURCE_STATUS.csv":
        "3f6922821b4d3f01ff8b84405584acea0d8270a5dba7f4402aaa7ee1fafc9ade",

    G2_SRC / "T26_CPU1_GPU_BATCH1_ANCHOR.csv":
        "cc3479a5dadf897fa8beb5eddc2a05ea7e4ed9548e7e7efe0aecc04f09efcfc8",

    G2_SRC / "F26_CPU1_GPU_P95_SPEEDUP.png":
        "a5b23cd1513d21798cad7cf1aef3c85e974bbe8086b6d1a868125d2b05b4ced7",

    G2_SRC / "F26_CPU1_GPU_P95_SPEEDUP.pdf":
        "a5fa474c6d260521b793e2c380166448e4eb7c055d3b93920601f18b865c44c5",

    G2_SRC / "F26_GPU_DELTA_PEAK_MEMORY.png":
        "b5848e0b9bed6c5393feec835c51c120a77bcb8bb7fa0ef1f60ecff10afab24b",

    G2_SRC / "F26_GPU_DELTA_PEAK_MEMORY.pdf":
        "55dae00f6eb49ed0c136832a383c9595dbd1790e0c98a986eab901656966ab8b",

    G2_SRC / "stage26_g2_final_gpu_publication_index.json":
        "8bb41ad37e3558014ff0d4209dc8d75f323a109ff94462b23fc9a9290462b39a",

    G2_SRC / "stage26_g2_final_stage26_closure_receipt.json":
        "e564bc9408b3bb61596e96442e604b1cee03277718015cfd78a0811a47871f65",

    G2_SRC / "stage26_g2_final_gpu_publication_closure_manifest.json":
        "1bc78ff5b9d5a4ebae17aa9b3350824da528a29c6e33b44729010cce17ce6bec",
}


for path, expected in EXPECTED_SHA256.items():

    actual = sha256_file(
        path
    )

    result = (
        "PASS"
        if actual == expected
        else "FAIL"
    )

    print(
        f"{result:4s}  "
        f"{path.name}"
    )

    if actual != expected:
        print(
            "  expected:",
            expected,
        )
        print(
            "  actual  :",
            actual,
        )

        raise RuntimeError(
            f"Frozen SHA256 mismatch: {path}"
        )


# =============================================================================
# 6. DESTINATION ABSENCE GATE
# =============================================================================

banner("STAGE26-PC0 :: MISSING CANONICAL DESTINATION GATE")


canonical_destinations = [
    BASE / "stage26_8_gpu_protocol_lock",
    BASE / "stage26_9_gpu_inference",
    BASE / "stage26_10_gpu_memory",
    BASE / "stage26_11_gpu_cpu_comparison",
    BASE / "stage26_12_final_synthesis",
    BASE / "stage26_publication_package",

    REPO / "docs" / "STAGE26_MANUSCRIPT_INTEGRATION.md",
    REPO / "docs" / "STAGE26_PUBLICATION_CLOSEOUT.md",

    REPO / "figures" / "stage26_deployment_profiling",
]


for path in canonical_destinations:

    exists = path.exists()

    print(
        f"{'EXISTS' if exists else 'MISSING':7s}",
        relative_repo(path),
    )

    if exists:
        raise RuntimeError(
            "Canonical destination unexpectedly already exists: "
            f"{path}"
        )


# =============================================================================
# 7. CLEAN TEMP BUILD
# =============================================================================

banner("STAGE26-PC0 :: INITIALIZE TEMPORARY CLOSEOUT BUILD")


if BUILD.exists():
    shutil.rmtree(
        BUILD
    )


BUILD.mkdir(
    parents=True,
    exist_ok=False,
)


provenance = []


# Repository-relative destination roots inside BUILD.
B_BASE = (
    BUILD
    / "results"
    / "stage26_deployment_profiling"
)

B_GPU_LOCK = (
    B_BASE
    / "stage26_8_gpu_protocol_lock"
)

B_GPU_INFERENCE = (
    B_BASE
    / "stage26_9_gpu_inference"
)

B_GPU_MEMORY = (
    B_BASE
    / "stage26_10_gpu_memory"
)

B_GPU_COMPARE = (
    B_BASE
    / "stage26_11_gpu_cpu_comparison"
)

B_FINAL_SYNTH = (
    B_BASE
    / "stage26_12_final_synthesis"
)

B_PUB = (
    B_BASE
    / "stage26_publication_package"
)

B_DOCS = (
    BUILD
    / "docs"
)

B_FIGURES = (
    BUILD
    / "figures"
    / "stage26_deployment_profiling"
)


for path in [
    B_GPU_LOCK,
    B_GPU_INFERENCE,
    B_GPU_MEMORY,
    B_GPU_COMPARE,
    B_FINAL_SYNTH,
    B_PUB / "tables",
    B_PUB / "figures",
    B_PUB / "manuscript_support",
    B_DOCS,
    B_FIGURES,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )


# =============================================================================
# 8. STAGE26_8_GPU_PROTOCOL_LOCK
#    POST-CLOSURE ARCHIVAL RECONSTRUCTION ONLY
# =============================================================================

banner("STAGE26-PC0 :: BUILD stage26_8_gpu_protocol_lock")


for name in [
    "stage26_g1_gpu_execution_plan.csv",
    "stage26_g1_gpu_environment.json",
    "stage26_g1_gpu_profile_receipt.json",
]:
    copy_verified(
        G1_SRC / name,
        B_GPU_LOCK / name,
        "GPU_PROTOCOL_ARCHIVAL_SOURCE",
        provenance,
    )


archival_notice = {
    "schema":
        "stage26_8_gpu_protocol_lock_archival_notice_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "directory_role":
        "POST_CLOSURE_CANONICAL_ARCHIVAL_INDEX",

    "prospective_preregistration":
        False,

    "retroactive_protocol_claim":
        False,

    "new_measurement":
        False,

    "gpu_rerun":
        False,

    "scientific_stage26_anchor":
        STAGE26_SCIENTIFIC_ANCHOR,

    "gpu_g1_anchor":
        STAGE26_GPU_G1_ANCHOR,

    "statement": (
        "This directory was created after Stage26 scientific closure to provide "
        "the canonical publication-facing GPU protocol location. It indexes and "
        "copies already-frozen G1 protocol/execution artifacts byte-for-byte. "
        "It must not be cited as evidence that this directory itself existed "
        "prospectively before GPU measurement."
    ),
}


write_text_file(
    B_GPU_LOCK
    / "stage26_8_gpu_protocol_lock_ARCHIVAL_NOTICE.json",

    json.dumps(
        archival_notice,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    ),
)


write_text_file(
    B_GPU_LOCK / "README.md",
    f"""
# Stage26-8 GPU Protocol Lock — Archival Reconstruction

**Scientific Stage26 anchor:** `{STAGE26_SCIENTIFIC_ANCHOR}`  
**Original durable G1 GPU anchor:** `{STAGE26_GPU_G1_ANCHOR}`

This directory is a **post-closure canonical archival reconstruction/index**.
It was created after Stage26 measurement science was already complete.

It does **not** claim that this directory existed prospectively before GPU
measurement.

The copied files are byte-identical frozen artifacts from
`stage26_g1_gpu_warm_profile/`:

- `stage26_g1_gpu_execution_plan.csv`
- `stage26_g1_gpu_environment.json`
- `stage26_g1_gpu_profile_receipt.json`

No measurement, inference, memory profiling, model loading, corpus access, or
GPU execution was performed while constructing this directory.
""",
)


# =============================================================================
# 9. STAGE26_9_GPU_INFERENCE
# =============================================================================

banner("STAGE26-PC0 :: BUILD stage26_9_gpu_inference")


for name in G1_REQUIRED:
    copy_verified(
        G1_SRC / name,
        B_GPU_INFERENCE / name,
        "FROZEN_G1_GPU_PROFILE",
        provenance,
    )


write_text_file(
    B_GPU_INFERENCE / "README.md",
    f"""
# Stage26-9 GPU Inference

Canonical publication-facing archive of the already-frozen Stage26 G1 GPU
profiling run.

**Original G1 durable anchor:** `{STAGE26_GPU_G1_ANCHOR}`

The seven G1 files in this directory are byte-for-byte copies of the original
committed GPU artifacts. No GPU measurement was rerun.
""",
)


# =============================================================================
# 10. STAGE26_10_GPU_MEMORY
# =============================================================================

banner("STAGE26-PC0 :: BUILD stage26_10_gpu_memory")


for name in [
    "T26_GPU_WARM_INFERENCE_MEMORY.csv",
    "T26_GPU_BACKEND_RESOURCE_STATUS.csv",
    "F26_GPU_DELTA_PEAK_MEMORY.png",
    "F26_GPU_DELTA_PEAK_MEMORY.pdf",
]:
    copy_verified(
        G2_SRC / name,
        B_GPU_MEMORY / name,
        "FROZEN_GPU_MEMORY_PUBLICATION_ARTIFACT",
        provenance,
    )


for name in [
    "stage26_g1_gpu_condition_summary.csv",
    "stage26_g1_gpu_worker_results.json",
]:
    copy_verified(
        G1_SRC / name,
        B_GPU_MEMORY / name,
        "FROZEN_GPU_MEMORY_PROVENANCE",
        provenance,
    )


write_text_file(
    B_GPU_MEMORY / "README.md",
    """
# Stage26-10 GPU Memory

Canonical archive for frozen GPU memory/resource evidence.

The publication table and figure are copied byte-for-byte from the final G2
closure package. G1 condition/worker artifacts are included for provenance.

`RESOURCE_LIMIT_OOM` and `BACKEND_UNAVAILABLE` remain observed outcomes.
No missing deployment cost is imputed.
""",
)


# =============================================================================
# 11. STAGE26_11_GPU_CPU_COMPARISON
# =============================================================================

banner("STAGE26-PC0 :: BUILD stage26_11_gpu_cpu_comparison")


for name in [
    "T26_CPU1_GPU_MATCHED_COMPARISON.csv",
    "T26_CPU1_GPU_BATCH1_ANCHOR.csv",
    "F26_CPU1_GPU_P95_SPEEDUP.png",
    "F26_CPU1_GPU_P95_SPEEDUP.pdf",
]:
    copy_verified(
        G2_SRC / name,
        B_GPU_COMPARE / name,
        "FROZEN_CPU_GPU_COMPARISON",
        provenance,
    )


write_text_file(
    B_GPU_COMPARE / "README.md",
    """
# Stage26-11 CPU/GPU Comparison

Canonical publication-facing archive of the frozen matched CPU1-versus-GPU
component-level comparison.

Interpretation constraints:

- matched PASS conditions only;
- component-level inference only;
- CPU/GPU ratio confidence intervals were not computed;
- no post-hoc optimal-batch claim;
- no missing-cost imputation.
""",
)


# =============================================================================
# 12. STAGE26_12_FINAL_SYNTHESIS
# =============================================================================

banner("STAGE26-PC0 :: BUILD stage26_12_final_synthesis")


# G2 final closure records
for name in [
    "stage26_g2_final_gpu_publication_index.json",
    "stage26_g2_final_stage26_closure_receipt.json",
    "stage26_g2_final_gpu_publication_closure_manifest.json",
]:
    copy_verified(
        G2_SRC / name,
        B_FINAL_SYNTH / name,
        "FINAL_STAGE26_CLOSURE_RECORD",
        provenance,
    )


# P2 derived narrative package already in repository.
for name in [
    "stage26_deployment_ieee_ready_narrative.md",
    "stage26_deployment_claim_traceability.csv",
    "stage26_deployment_manuscript_asset_map.csv",
    "stage26_p2_narrative_receipt.json",
    "stage26_p2_narrative_manifest.json",
]:
    copy_verified(
        P2_SRC / name,
        B_FINAL_SYNTH / name,
        "DERIVED_PAPER_FACING_SYNTHESIS",
        provenance,
    )


# Original P1 preserved draft.
for name in [
    "stage26_results_deployment_discussion_draft.md",
    "stage26_claim_traceability.csv",
    "stage26_manuscript_asset_plan.csv",
]:
    copy_verified(
        P1_SRC / name,
        B_FINAL_SYNTH / name,
        "ORIGINAL_P1_MANUSCRIPT_SUPPORT",
        provenance,
    )


write_text_file(
    B_FINAL_SYNTH / "README.md",
    f"""
# Stage26-12 Final Synthesis

**Scientific closure anchor:** `{STAGE26_SCIENTIFIC_ANCHOR}`

This directory consolidates already-existing Stage26 closure records and
paper-facing narrative support.

It introduces **no new science**.

The original P1 manuscript draft is retained alongside the later derived P2
narrative so provenance is explicit rather than silently replacing one with the
other.
""",
)


# =============================================================================
# 13. BUILD CANONICAL PUBLICATION TABLE PACKAGE
# =============================================================================

banner("STAGE26-PC0 :: BUILD PUBLICATION TABLE PACKAGE")


for src in cpu_tables:
    copy_verified(
        src,
        B_PUB / "tables" / src.name,
        "CPU_PUBLICATION_TABLE",
        provenance,
    )


for name in [
    "T26_GPU_WARM_INFERENCE_MEMORY.csv",
    "T26_CPU1_GPU_MATCHED_COMPARISON.csv",
    "T26_GPU_BACKEND_RESOURCE_STATUS.csv",
    "T26_CPU1_GPU_BATCH1_ANCHOR.csv",
]:
    copy_verified(
        G2_SRC / name,
        B_PUB / "tables" / name,
        "GPU_OR_CPU_GPU_PUBLICATION_TABLE",
        provenance,
    )


publication_tables = sorted(
    (
        B_PUB
        / "tables"
    ).glob(
        "T26_*.csv"
    )
)


print(
    "Publication tables:",
    len(publication_tables),
)


if len(publication_tables) != 12:
    raise RuntimeError(
        f"Expected 12 publication tables, got {len(publication_tables)}"
    )


# =============================================================================
# 14. BUILD CANONICAL PUBLICATION FIGURE PACKAGE
# =============================================================================

banner("STAGE26-PC0 :: BUILD PUBLICATION FIGURE PACKAGE")


for src in cpu_pngs + cpu_pdfs:
    copy_verified(
        src,
        B_PUB / "figures" / src.name,
        "CPU_PUBLICATION_FIGURE",
        provenance,
    )


for name in [
    "F26_CPU1_GPU_P95_SPEEDUP.png",
    "F26_CPU1_GPU_P95_SPEEDUP.pdf",
    "F26_GPU_DELTA_PEAK_MEMORY.png",
    "F26_GPU_DELTA_PEAK_MEMORY.pdf",
]:
    copy_verified(
        G2_SRC / name,
        B_PUB / "figures" / name,
        "GPU_PUBLICATION_FIGURE",
        provenance,
    )


publication_pngs = sorted(
    (
        B_PUB
        / "figures"
    ).glob(
        "F26_*.png"
    )
)

publication_pdfs = sorted(
    (
        B_PUB
        / "figures"
    ).glob(
        "F26_*.pdf"
    )
)


print(
    "Publication PNGs:",
    len(publication_pngs),
)

print(
    "Publication PDFs:",
    len(publication_pdfs),
)


if len(publication_pngs) != 10:
    raise RuntimeError(
        f"Expected 10 publication PNGs, got {len(publication_pngs)}"
    )

if len(publication_pdfs) != 10:
    raise RuntimeError(
        f"Expected 10 publication PDFs, got {len(publication_pdfs)}"
    )


# =============================================================================
# 15. TOP-LEVEL figures/stage26_deployment_profiling/
# =============================================================================

banner("STAGE26-PC0 :: BUILD TOP-LEVEL STAGE26 FIGURE DIRECTORY")


for src in publication_pngs + publication_pdfs:

    # publication figures currently live under BUILD, not REPO, so copy directly
    dst = (
        B_FIGURES
        / src.name
    )

    shutil.copyfile(
        src,
        dst,
    )

    if sha256_file(
        src
    ) != sha256_file(
        dst
    ):
        raise RuntimeError(
            f"Top-level figure byte mismatch: {src.name}"
        )


print(
    "Top-level Stage26 figure files:",
    len(
        list(
            B_FIGURES.glob(
                "F26_*"
            )
        )
    ),
)


# =============================================================================
# 16. BUILD MANUSCRIPT SUPPORT PACKAGE
# =============================================================================

banner("STAGE26-PC0 :: BUILD PUBLICATION MANUSCRIPT SUPPORT")


for name in [
    "stage26_results_deployment_discussion_draft.md",
    "stage26_claim_traceability.csv",
    "stage26_manuscript_asset_plan.csv",
]:
    copy_verified(
        P1_SRC / name,
        B_PUB / "manuscript_support" / name,
        "ORIGINAL_P1_MANUSCRIPT_SUPPORT",
        provenance,
    )


for name in [
    "stage26_deployment_ieee_ready_narrative.md",
    "stage26_deployment_claim_traceability.csv",
    "stage26_deployment_manuscript_asset_map.csv",
]:
    copy_verified(
        P2_SRC / name,
        B_PUB / "manuscript_support" / name,
        "DERIVED_P2_MANUSCRIPT_SUPPORT",
        provenance,
    )


# =============================================================================
# 17. DOCS/STAGE26_MANUSCRIPT_INTEGRATION.md
# =============================================================================

banner("STAGE26-PC0 :: BUILD MANUSCRIPT INTEGRATION DOCUMENT")


integration_doc = f"""
# Stage26 Manuscript Integration

## Scientific identity

- Final Stage26 scientific anchor:
  `{STAGE26_SCIENTIFIC_ANCHOR}`
- CPU closure anchor:
  `{STAGE26_CPU_CLOSURE_ANCHOR}`
- G1 GPU profiling anchor:
  `{STAGE26_GPU_G1_ANCHOR}`

Stage26 deployment profiling is complete and frozen. This document is a
post-closure manuscript integration guide and does not introduce new
measurements.

## Main-paper assets

Recommended core deployment assets:

1. `F26_CPU1_GPU_P95_SPEEDUP`
   - primary CPU1/GPU deployment comparison;
   - demonstrates architecture- and batch-dependent acceleration.

2. `T26_CPU1_GPU_BATCH1_ANCHOR`
   - low-batch / online inference anchor.

3. `F26_PARETO`
   - predictive-performance versus CPU1 batch-1 latency;
   - Group A and Group B must remain scientifically separate.

4. `F26_GPU_DELTA_PEAK_MEMORY`
   - GPU memory/resource feasibility.

## Supplementary deployment assets

The canonical publication package also contains:

- complete CPU warm-inference table;
- CPU cold-start table;
- CPU memory/package table;
- component measurement table;
- capacity/scaling table;
- representation sensitivity table;
- Group-B representation/inference-ratio table;
- full matched CPU1/GPU comparison table;
- GPU warm inference/memory table;
- GPU backend/resource-status table;
- all CPU and GPU deployment figure families in PNG and PDF form.

## Claim boundaries

The following restrictions are mandatory:

1. Use **component-level inference**, not complete end-to-end IDS latency.
2. Complete extraction-to-inference E2E measurement is unavailable.
3. Do not compare Pareto frontiers across Group A and Group B.
4. CPU/GPU speedup ratios are descriptive point estimates; ratio-level
   confidence intervals were not computed.
5. Do not impute cost for `RESOURCE_LIMIT_OOM`, timeout, or
   `BACKEND_UNAVAILABLE`.
6. Do not describe frozen batch sizes as retrospectively optimized.
7. LightGBM GPU backend unavailability is not a latency result.
8. The single-T4 results are hardware-specific.

## Canonical publication locations

Tables:

`results/stage26_deployment_profiling/stage26_publication_package/tables/`

Figures:

`results/stage26_deployment_profiling/stage26_publication_package/figures/`

Top-level figure mirror:

`figures/stage26_deployment_profiling/`

Final synthesis:

`results/stage26_deployment_profiling/stage26_12_final_synthesis/`
"""


write_text_file(
    B_DOCS
    / "STAGE26_MANUSCRIPT_INTEGRATION.md",

    integration_doc,
)


# =============================================================================
# 18. DOCS/STAGE26_PUBLICATION_CLOSEOUT.md
# =============================================================================

banner("STAGE26-PC0 :: BUILD PUBLICATION CLOSEOUT DOCUMENT")


closeout_doc = f"""
# Stage26 Publication Closeout

## Status

**STAGE26: COMPLETE / CLOSED**

Scientific Stage26 closure remains anchored at:

`{STAGE26_SCIENTIFIC_ANCHOR}`

This publication-closeout pass reorganizes and mirrors already-frozen artifacts
for manuscript use. It does not reopen Stage26 science.

## Canonical archival structure

The following publication-facing directories are established:

- `stage26_8_gpu_protocol_lock/`
- `stage26_9_gpu_inference/`
- `stage26_10_gpu_memory/`
- `stage26_11_gpu_cpu_comparison/`
- `stage26_12_final_synthesis/`
- `stage26_publication_package/`

### Important archival qualification

`stage26_8_gpu_protocol_lock/` is a **post-closure archival reconstruction and
canonical index**.

It must **not** be described as a directory that existed prospectively before
GPU profiling.

The underlying execution plan, environment, receipt, timing observations,
memory/resource outcomes, and final G2 closure artifacts were already frozen
and committed during the original Stage26 scientific workflow.

## Publication package geometry

- CPU publication tables: 8
- GPU / CPU-GPU publication tables: 4
- Total canonical publication tables: 12

- CPU figure families: 8
- GPU figure families: 2
- Total figure families: 10
- PNG files: 10
- PDF files: 10

## Scientific safeguards

- New GPU execution: **NONE**
- New CPU inference: **NONE**
- New timing: **NONE**
- New memory profiling: **NONE**
- Model loading: **NONE**
- Corpus access: **NONE**
- Retraining: **NONE**
- Complete E2E claim: **NONE**
- Cross-group Pareto: **NONE**
- Ratio confidence intervals invented: **NO**
- Missing deployment cost imputed: **NO**

## Frozen source anchors

- CPU closure:
  `{STAGE26_CPU_CLOSURE_ANCHOR}`
- G1 GPU profiling:
  `{STAGE26_GPU_G1_ANCHOR}`
- Final CPU/GPU scientific closure:
  `{STAGE26_SCIENTIFIC_ANCHOR}`

Stage26 should not be rerun for publication packaging.
"""


write_text_file(
    B_DOCS
    / "STAGE26_PUBLICATION_CLOSEOUT.md",

    closeout_doc,
)


# Copy docs into publication package too.
for name in [
    "STAGE26_MANUSCRIPT_INTEGRATION.md",
    "STAGE26_PUBLICATION_CLOSEOUT.md",
]:

    src = B_DOCS / name
    dst = (
        B_PUB
        / "manuscript_support"
        / name
    )

    shutil.copyfile(
        src,
        dst,
    )

    if sha256_file(
        src
    ) != sha256_file(
        dst
    ):
        raise RuntimeError(
            f"Document package copy mismatch: {name}"
        )


# =============================================================================
# 19. PUBLICATION PACKAGE README
# =============================================================================

banner("STAGE26-PC0 :: BUILD PUBLICATION PACKAGE README")


write_text_file(
    B_PUB / "README.md",
    f"""
# Stage26 Publication Package

Scientific source anchor:

`{STAGE26_SCIENTIFIC_ANCHOR}`

This package consolidates already-frozen Stage26 deployment artifacts.

Contents:

- `tables/` — 12 publication tables
- `figures/` — 10 PNG + 10 PDF publication figures
- `manuscript_support/` — integration, closeout, traceability, and narrative
  support

No experiment was rerun to construct this package.
""",
)


# =============================================================================
# 20. PACKAGE MANIFESTS
# =============================================================================

banner("STAGE26-PC0 :: BUILD PACKAGE MANIFESTS")


package_files = sorted(
    p
    for p in B_PUB.rglob(
        "*"
    )
    if p.is_file()
)


manifest_rows = []


for path in package_files:

    # Manifest files are not created yet, so no self-reference issue.
    manifest_rows.append(
        {
            "path":
                str(
                    path.relative_to(
                        B_PUB
                    )
                ),

            "bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest_csv = (
    B_PUB
    / "stage26_publication_package_manifest.csv"
)


with manifest_csv.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "path",
            "bytes",
            "sha256",
        ],
    )

    writer.writeheader()
    writer.writerows(
        manifest_rows
    )


manifest_json = {
    "schema":
        "stage26_publication_package_manifest_v1",

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_stage26_anchor":
        STAGE26_SCIENTIFIC_ANCHOR,

    "packaging_parent_head":
        EXPECTED_CURRENT_HEAD,

    "new_science":
        False,

    "gpu_rerun":
        False,

    "cpu_rerun":
        False,

    "table_count":
        12,

    "figure_family_count":
        10,

    "png_count":
        10,

    "pdf_count":
        10,

    "files":
        manifest_rows,
}


write_text_file(
    B_PUB
    / "stage26_publication_package_manifest.json",

    json.dumps(
        manifest_json,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    ),
)


# =============================================================================
# 21. FULL CLOSEOUT PROVENANCE MANIFEST
# =============================================================================

banner("STAGE26-PC0 :: BUILD FULL CLOSEOUT PROVENANCE")


prov_csv = (
    B_FINAL_SYNTH
    / "stage26_closeout_copy_provenance.csv"
)


with prov_csv.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "role",
            "source_repository_path",
            "destination_repository_path",
            "bytes",
            "sha256",
            "copy_mode",
        ],
    )

    writer.writeheader()

    writer.writerows(
        provenance
    )


closeout_manifest = {
    "schema":
        "stage26_canonical_publication_closeout_manifest_v1",

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_stage26_anchor":
        STAGE26_SCIENTIFIC_ANCHOR,

    "cpu_closure_anchor":
        STAGE26_CPU_CLOSURE_ANCHOR,

    "gpu_g1_anchor":
        STAGE26_GPU_G1_ANCHOR,

    "repository_parent":
        EXPECTED_CURRENT_HEAD,

    "stage26_8_gpu_protocol_lock_status":
        "POST_CLOSURE_ARCHIVAL_RECONSTRUCTION_NOT_PROSPECTIVE",

    "new_measurement":
        False,

    "gpu_used":
        False,

    "model_loaded":
        False,

    "corpus_accessed":
        False,

    "publication_tables":
        12,

    "publication_pngs":
        10,

    "publication_pdfs":
        10,

    "byte_for_byte_copy_records":
        len(
            provenance
        ),
}


write_text_file(
    B_FINAL_SYNTH
    / "stage26_canonical_publication_closeout_manifest.json",

    json.dumps(
        closeout_manifest,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    ),
)


# =============================================================================
# 22. AUDIT TEMPORARY BUILD BEFORE TOUCHING REPOSITORY
# =============================================================================

banner("STAGE26-PC0 :: TEMP BUILD AUDIT")


all_build_files = sorted(
    p
    for p in BUILD.rglob(
        "*"
    )
    if p.is_file()
)


print(
    "Temporary build files:",
    len(
        all_build_files
    )
)

print()


for path in all_build_files:

    print(
        f"{human_bytes(path.stat().st_size):>12s}  "
        f"{path.relative_to(BUILD)}"
    )


# Required canonical structures.
required_build_paths = [
    B_GPU_LOCK,
    B_GPU_INFERENCE,
    B_GPU_MEMORY,
    B_GPU_COMPARE,
    B_FINAL_SYNTH,
    B_PUB,
    B_DOCS / "STAGE26_MANUSCRIPT_INTEGRATION.md",
    B_DOCS / "STAGE26_PUBLICATION_CLOSEOUT.md",
    B_FIGURES,
]


for path in required_build_paths:

    if not path.exists():
        raise RuntimeError(
            f"Temporary canonical build missing: {path}"
        )


# Publication geometry final check.
if len(
    list(
        (B_PUB / "tables").glob(
            "T26_*.csv"
        )
    )
) != 12:
    raise RuntimeError(
        "Final package does not contain exactly 12 tables."
    )


if len(
    list(
        (B_PUB / "figures").glob(
            "F26_*.png"
        )
    )
) != 10:
    raise RuntimeError(
        "Final package does not contain exactly 10 PNG figures."
    )


if len(
    list(
        (B_PUB / "figures").glob(
            "F26_*.pdf"
        )
    )
) != 10:
    raise RuntimeError(
        "Final package does not contain exactly 10 PDF figures."
    )


# =============================================================================
# 23. INSTALL TEMP BUILD INTO REPOSITORY
# =============================================================================

banner("STAGE26-PC0 :: INSTALL CANONICAL BUILD INTO REPOSITORY")


install_pairs = [
    (
        B_GPU_LOCK,
        BASE / "stage26_8_gpu_protocol_lock",
    ),
    (
        B_GPU_INFERENCE,
        BASE / "stage26_9_gpu_inference",
    ),
    (
        B_GPU_MEMORY,
        BASE / "stage26_10_gpu_memory",
    ),
    (
        B_GPU_COMPARE,
        BASE / "stage26_11_gpu_cpu_comparison",
    ),
    (
        B_FINAL_SYNTH,
        BASE / "stage26_12_final_synthesis",
    ),
    (
        B_PUB,
        BASE / "stage26_publication_package",
    ),
    (
        B_FIGURES,
        REPO
        / "figures"
        / "stage26_deployment_profiling",
    ),
]


for src, dst in install_pairs:

    if dst.exists():
        raise RuntimeError(
            f"Destination appeared during build: {dst}"
        )

    shutil.copytree(
        src,
        dst,
    )

    print(
        "INSTALLED",
        relative_repo(
            dst
        ),
    )


for name in [
    "STAGE26_MANUSCRIPT_INTEGRATION.md",
    "STAGE26_PUBLICATION_CLOSEOUT.md",
]:

    src = (
        B_DOCS
        / name
    )

    dst = (
        REPO
        / "docs"
        / name
    )

    if dst.exists():
        raise RuntimeError(
            f"Destination appeared during build: {dst}"
        )

    shutil.copyfile(
        src,
        dst,
    )

    print(
        "INSTALLED",
        relative_repo(
            dst
        ),
    )


# =============================================================================
# 24. POST-INSTALL BYTE AUDIT
# =============================================================================

banner("STAGE26-PC0 :: POST-INSTALL BYTE AUDIT")


for src, dst in install_pairs:

    src_files = sorted(
        p
        for p in src.rglob(
            "*"
        )
        if p.is_file()
    )

    dst_files = sorted(
        p
        for p in dst.rglob(
            "*"
        )
        if p.is_file()
    )


    src_rel = [
        str(
            p.relative_to(
                src
            )
        )
        for p in src_files
    ]

    dst_rel = [
        str(
            p.relative_to(
                dst
            )
        )
        for p in dst_files
    ]


    if src_rel != dst_rel:
        raise RuntimeError(
            f"Installed file-set mismatch: {dst}"
        )


    for a, b in zip(
        src_files,
        dst_files,
    ):

        if sha256_file(
            a
        ) != sha256_file(
            b
        ):

            raise RuntimeError(
                f"Installed byte mismatch: {b}"
            )


    print(
        f"PASS {relative_repo(dst)} "
        f"({len(dst_files)} files)"
    )


for name in [
    "STAGE26_MANUSCRIPT_INTEGRATION.md",
    "STAGE26_PUBLICATION_CLOSEOUT.md",
]:

    a = (
        B_DOCS
        / name
    )

    b = (
        REPO
        / "docs"
        / name
    )

    if sha256_file(
        a
    ) != sha256_file(
        b
    ):
        raise RuntimeError(
            f"Installed doc byte mismatch: {name}"
        )


    print(
        "PASS",
        relative_repo(
            b
        ),
    )


# =============================================================================
# 25. GIT STATUS — DO NOT STAGE OR COMMIT YET
# =============================================================================

banner("STAGE26-PC0 COMPLETE :: CANONICAL CLOSEOUT BUILT LOCALLY")


git_status = git(
    "status",
    "--short",
)


print(git_status)


print()
print(
    "Scientific Stage26 anchor :",
    STAGE26_SCIENTIFIC_ANCHOR,
)

print(
    "Repository parent         :",
    EXPECTED_CURRENT_HEAD,
)

print()
print(
    "GPU used                  : NO"
)

print(
    "New inference             : NO"
)

print(
    "New timing                : NO"
)

print(
    "New memory profiling      : NO"
)

print(
    "Model loading             : NO"
)

print(
    "Corpus access             : NO"
)

print(
    "New scientific result     : NO"
)

print()
print(
    "Canonical tables          : 12"
)

print(
    "Canonical figure families : 10"
)

print(
    "Canonical PNGs            : 10"
)

print(
    "Canonical PDFs            : 10"
)

print()
print(
    "stage26_8_gpu_protocol_lock:"
)

print(
    "  POST-CLOSURE ARCHIVAL RECONSTRUCTION / INDEX"
)

print(
    "  NOT represented as prospective preregistration"
)

print()
print(
    "COMMIT/PUSH PERFORMED      : NO"
)

print()
print(
    "NEXT:"
)

print(
    "  Paste the COMPLETE output."
)

print(
    "  After we verify the local file set, the next single cell will "
    "stage exactly these closeout artifacts, commit, push, and remotely "
    "byte-verify the publication package."
)


STAGE26-PC0 :: REPOSITORY / HISTORY GATE
Expected current HEAD    : 3bf4c6e9e9cb2e6ccfeb108e35cbf5fbeded6fd4
Local HEAD               : 3bf4c6e9e9cb2e6ccfeb108e35cbf5fbeded6fd4
origin/main              : 3bf4c6e9e9cb2e6ccfeb108e35cbf5fbeded6fd4
Repository clean         : True

Stage26 scientific anchor: 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
Stage26 CPU anchor       : 304e5613627a744cbd5d369857f8ac5667a520eb
Stage26 G1 GPU anchor    : 79b9c23c91ef7185d35b700e6f6f84aefb54ab35
Stage26 scientific anchor     : True
Stage26 CPU closure anchor    : True
Stage26 G1 GPU anchor         : True

STAGE26-PC0 :: VERIFY FROZEN SOURCE DIRECTORIES
PASS results/stage26_deployment_profiling/stage26_8d3_cpu_publication_tables
PASS results/stage26_deployment_profiling/stage26_8d4_cpu_publication_figures
PASS results/stage26_deployment_profiling/stage26_g1_gpu_warm_profile
PASS results/stage26_deployment_profiling/stage26_g2_final_gpu_publication_closure
PASS results/stage26_deployment_profiling/stage26

In [10]:
# =============================================================================
# STAGE26-PC1 :: COMMIT / PUSH / REMOTE-VERIFY CANONICAL PUBLICATION CLOSEOUT
#
# PRECONDITION:
#   Stage26-PC0 completed successfully.
#
# THIS CELL:
#   1. verifies the exact 104-file local closeout package,
#   2. stages ONLY those files,
#   3. commits on top of 3bf4c6e...,
#   4. pushes main securely,
#   5. verifies all 104 files byte-for-byte from origin/main,
#   6. declares the canonical Stage26 publication closeout durable.
#
# NO GPU
# NO inference
# NO timing
# NO memory profiling
# NO model loading
# NO corpus access
# NO new science
# =============================================================================

from __future__ import annotations

import hashlib
import os
import stat
import subprocess
import tempfile
from pathlib import Path


# =============================================================================
# 0. IDENTITIES
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "3bf4c6e9e9cb2e6ccfeb108e35cbf5fbeded6fd4"
)

STAGE26_SCIENTIFIC_ANCHOR = (
    "9e8354ecc9cfa72c28aa037e5d2053de422bf7a2"
)

COMMIT_MSG = (
    "stage26: add canonical publication closeout package"
)

BASE = (
    REPO
    / "results"
    / "stage26_deployment_profiling"
)


CANONICAL_ROOTS = [
    REPO
    / "docs"
    / "STAGE26_MANUSCRIPT_INTEGRATION.md",

    REPO
    / "docs"
    / "STAGE26_PUBLICATION_CLOSEOUT.md",

    REPO
    / "figures"
    / "stage26_deployment_profiling",

    BASE
    / "stage26_8_gpu_protocol_lock",

    BASE
    / "stage26_9_gpu_inference",

    BASE
    / "stage26_10_gpu_memory",

    BASE
    / "stage26_11_gpu_cpu_comparison",

    BASE
    / "stage26_12_final_synthesis",

    BASE
    / "stage26_publication_package",
]


# =============================================================================
# HELPERS
# =============================================================================

def banner(text):
    print()
    print("=" * 124)
    print(text)
    print("=" * 124)


def run(cmd, *, cwd=REPO, env=None, check=True):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(map(str, cmd))}\n\n"
            f"{p.stdout}"
        )

    return p.stdout.strip()


def git(*args, env=None, check=True):
    return run(
        ["git", *args],
        env=env,
        check=check,
    )


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for block in iter(
            lambda: f.read(16 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def repo_rel(path):
    return str(
        Path(path).relative_to(REPO)
    )


# =============================================================================
# 1. PARENT / HISTORY GATE
# =============================================================================

banner("STAGE26-PC1 :: PARENT / HISTORY GATE")

git(
    "fetch",
    "origin",
    "main",
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)


print("Expected parent           :", EXPECTED_PARENT)
print("Local HEAD                :", head)
print("origin/main               :", origin)
print("Stage26 scientific anchor :", STAGE26_SCIENTIFIC_ANCHOR)


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Unexpected local HEAD before closeout commit."
    )

if origin != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main moved before closeout commit."
    )


rc = subprocess.run(
    [
        "git",
        "merge-base",
        "--is-ancestor",
        STAGE26_SCIENTIFIC_ANCHOR,
        head,
    ],
    cwd=REPO,
    check=False,
).returncode


print(
    "Scientific anchor ancestor:",
    rc == 0,
)


if rc != 0:
    raise RuntimeError(
        "Frozen Stage26 scientific anchor is not in current history."
    )


# =============================================================================
# 2. BUILD EXACT EXPECTED FILE SET
# =============================================================================

banner("STAGE26-PC1 :: EXACT 104-FILE PACKAGE AUDIT")


expected_files = []


for root in CANONICAL_ROOTS:

    if not root.exists():
        raise FileNotFoundError(
            root
        )

    if root.is_file():

        expected_files.append(
            repo_rel(root)
        )

    else:

        for path in root.rglob("*"):

            if path.is_file():

                expected_files.append(
                    repo_rel(path)
                )


expected_files = sorted(
    set(
        expected_files
    )
)


print(
    "Expected canonical files:",
    len(expected_files),
)


if len(expected_files) != 104:
    raise RuntimeError(
        f"Expected 104 canonical closeout files, "
        f"found {len(expected_files)}."
    )


# Check that ALL untracked files are exactly these closeout files.
untracked = sorted(
    x
    for x in git(
        "ls-files",
        "--others",
        "--exclude-standard",
    ).splitlines()
    if x
)


print(
    "Current untracked files  :",
    len(untracked),
)


if untracked != expected_files:

    only_expected = sorted(
        set(expected_files)
        - set(untracked)
    )

    unexpected = sorted(
        set(untracked)
        - set(expected_files)
    )

    print()
    print(
        "Expected but not untracked:",
        only_expected,
    )

    print()
    print(
        "Unexpected untracked:",
        unexpected,
    )

    raise RuntimeError(
        "Untracked file set does not exactly match "
        "the Stage26 closeout package."
    )


# There must be no tracked modifications before staging.
tracked_diff = git(
    "diff",
    "--name-only",
)


if tracked_diff:
    raise RuntimeError(
        "Unexpected tracked modifications before staging:\n"
        + tracked_diff
    )


# Record local SHA256 before Git touches anything.
local_sha = {
    rel:
        sha256_file(
            REPO / rel
        )
    for rel in expected_files
}


print(
    "Pre-stage SHA256 inventory:",
    len(local_sha),
    "/ 104",
)


# =============================================================================
# 3. PUBLICATION GEOMETRY CHECK
# =============================================================================

banner("STAGE26-PC1 :: PUBLICATION GEOMETRY CHECK")


PUB = (
    BASE
    / "stage26_publication_package"
)


tables = sorted(
    (
        PUB / "tables"
    ).glob(
        "T26_*.csv"
    )
)

pngs = sorted(
    (
        PUB / "figures"
    ).glob(
        "F26_*.png"
    )
)

pdfs = sorted(
    (
        PUB / "figures"
    ).glob(
        "F26_*.pdf"
    )
)

top_pngs = sorted(
    (
        REPO
        / "figures"
        / "stage26_deployment_profiling"
    ).glob(
        "F26_*.png"
    )
)

top_pdfs = sorted(
    (
        REPO
        / "figures"
        / "stage26_deployment_profiling"
    ).glob(
        "F26_*.pdf"
    )
)


print("Canonical tables              :", len(tables))
print("Publication-package PNGs      :", len(pngs))
print("Publication-package PDFs      :", len(pdfs))
print("Top-level Stage26 PNGs        :", len(top_pngs))
print("Top-level Stage26 PDFs        :", len(top_pdfs))


if len(tables) != 12:
    raise RuntimeError(
        "Expected exactly 12 canonical Stage26 tables."
    )

if len(pngs) != 10 or len(pdfs) != 10:
    raise RuntimeError(
        "Expected 10 PNG + 10 PDF figures in publication package."
    )

if len(top_pngs) != 10 or len(top_pdfs) != 10:
    raise RuntimeError(
        "Expected 10 PNG + 10 PDF figures in top-level Stage26 figure mirror."
    )


# Ensure mirrored figures are byte-identical.
for source in pngs + pdfs:

    mirror = (
        REPO
        / "figures"
        / "stage26_deployment_profiling"
        / source.name
    )

    if not mirror.is_file():
        raise FileNotFoundError(
            mirror
        )

    if sha256_file(source) != sha256_file(mirror):
        raise RuntimeError(
            f"Figure mirror differs: {source.name}"
        )


print(
    "Publication ↔ top-level figure byte identity: PASS"
)


# =============================================================================
# 4. ARCHIVAL QUALIFICATION CHECK
# =============================================================================

banner("STAGE26-PC1 :: GPU PROTOCOL ARCHIVAL QUALIFICATION")


notice = (
    BASE
    / "stage26_8_gpu_protocol_lock"
    / "stage26_8_gpu_protocol_lock_ARCHIVAL_NOTICE.json"
)


if not notice.is_file():
    raise FileNotFoundError(
        notice
    )


import json


notice_obj = json.loads(
    notice.read_text(
        encoding="utf-8"
    )
)


print(
    "directory_role             :",
    notice_obj.get(
        "directory_role"
    ),
)

print(
    "prospective_preregistration:",
    notice_obj.get(
        "prospective_preregistration"
    ),
)

print(
    "retroactive_protocol_claim :",
    notice_obj.get(
        "retroactive_protocol_claim"
    ),
)

print(
    "new_measurement            :",
    notice_obj.get(
        "new_measurement"
    ),
)


if (
    notice_obj.get(
        "directory_role"
    )
    !=
    "POST_CLOSURE_CANONICAL_ARCHIVAL_INDEX"
):
    raise RuntimeError(
        "Archival directory role is incorrect."
    )


if notice_obj.get(
    "prospective_preregistration"
) is not False:
    raise RuntimeError(
        "Archival reconstruction must not claim prospective preregistration."
    )


if notice_obj.get(
    "retroactive_protocol_claim"
) is not False:
    raise RuntimeError(
        "Archival reconstruction must not make a retroactive protocol claim."
    )


if notice_obj.get(
    "new_measurement"
) is not False:
    raise RuntimeError(
        "Closeout package unexpectedly claims new measurement."
    )


print()
print(
    "Archival qualification audit: PASS"
)


# =============================================================================
# 5. STAGE EXACTLY THE CANONICAL FILE SET
# =============================================================================

banner("STAGE26-PC1 :: STAGE EXACT CANONICAL FILE SET")


git(
    "add",
    "--",
    *expected_files,
)


staged = sorted(
    x
    for x in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if x
)


print(
    "Expected staged:",
    len(expected_files),
)

print(
    "Actual staged  :",
    len(staged),
)


if staged != expected_files:

    unexpected = sorted(
        set(staged)
        - set(expected_files)
    )

    missing = sorted(
        set(expected_files)
        - set(staged)
    )

    print(
        "Unexpected staged:",
        unexpected,
    )

    print(
        "Missing staged   :",
        missing,
    )

    raise RuntimeError(
        "Staged set is not exactly the canonical Stage26 package."
    )


# No unstaged tracked modifications.
unstaged_tracked = git(
    "diff",
    "--name-only",
)


if unstaged_tracked:
    raise RuntimeError(
        "Unexpected unstaged tracked files:\n"
        + unstaged_tracked
    )


# No remaining untracked files.
remaining_untracked = git(
    "ls-files",
    "--others",
    "--exclude-standard",
)


if remaining_untracked:
    raise RuntimeError(
        "Unexpected remaining untracked files:\n"
        + remaining_untracked
    )


print(
    "Exact staged file-set audit: PASS"
)


# =============================================================================
# 6. VERIFY INDEX BYTES BEFORE COMMIT
# =============================================================================

banner("STAGE26-PC1 :: PRE-COMMIT BYTE AUDIT")


for rel in expected_files:

    current = sha256_file(
        REPO / rel
    )

    if current != local_sha[rel]:
        raise RuntimeError(
            f"Local byte mutation detected before commit: {rel}"
        )


print(
    "All 104 local SHA256 values unchanged: PASS"
)


# =============================================================================
# 7. COMMIT
# =============================================================================

banner("STAGE26-PC1 :: COMMIT")


git(
    "commit",
    "-m",
    COMMIT_MSG,
)


new_head = git(
    "rev-parse",
    "HEAD",
)

parent = git(
    "rev-parse",
    "HEAD^",
)

subject = git(
    "show",
    "-s",
    "--format=%s",
    new_head,
)


print("Parent   :", parent)
print("New HEAD :", new_head)
print("Subject  :", subject)


if parent != EXPECTED_PARENT:
    raise RuntimeError(
        "Closeout commit has unexpected parent."
    )


if subject != COMMIT_MSG:
    raise RuntimeError(
        "Closeout commit subject mismatch."
    )


# =============================================================================
# 8. PUSH SECURELY
# =============================================================================

banner("STAGE26-PC1 :: PUSH")


from kaggle_secrets import UserSecretsClient


token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not token:
    raise RuntimeError(
        "GITHUB_TOKEN unavailable."
    )


fd, askpass_name = tempfile.mkstemp(
    prefix="stage26_pc1_askpass_",
    suffix=".sh",
)

os.close(fd)

askpass = Path(
    askpass_name
)


try:

    askpass.write_text(
        "#!/bin/sh\n"
        "case \"$1\" in\n"
        "  *Username*) printf '%s\\n' 'x-access-token' ;;\n"
        "  *Password*) printf '%s\\n' \"$TOKEN_VALUE\" ;;\n"
        "  *) printf '%s\\n' '' ;;\n"
        "esac\n",
        encoding="utf-8",
    )

    askpass.chmod(
        askpass.stat().st_mode
        | stat.S_IXUSR
    )


    env = os.environ.copy()

    env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    env[
        "TOKEN_VALUE"
    ] = token


    push_output = git(
        "push",
        "origin",
        "main",
        env=env,
    )


    print(
        push_output
    )


finally:

    askpass.unlink(
        missing_ok=True
    )

    token = None


# =============================================================================
# 9. REMOTE HEAD VERIFICATION
# =============================================================================

banner("STAGE26-PC1 :: REMOTE HEAD VERIFICATION")


git(
    "fetch",
    "origin",
    "main",
)


local_head = git(
    "rev-parse",
    "HEAD",
)

remote_head = git(
    "rev-parse",
    "origin/main",
)


print("Local HEAD :", local_head)
print("origin/main:", remote_head)


if not (
    local_head
    ==
    remote_head
    ==
    new_head
):
    raise RuntimeError(
        "Local/remote HEAD mismatch after push."
    )


# =============================================================================
# 10. VERIFY EXACT REMOTE FILE SET
# =============================================================================

banner("STAGE26-PC1 :: REMOTE FILE-SET VERIFICATION")


remote_tree = set(
    x
    for x in git(
        "ls-tree",
        "-r",
        "--name-only",
        "origin/main",
    ).splitlines()
    if x
)


missing_remote = [
    rel
    for rel in expected_files
    if rel not in remote_tree
]


print(
    "Expected canonical files:",
    len(expected_files),
)

print(
    "Missing remotely         :",
    len(missing_remote),
)


if missing_remote:

    for rel in missing_remote:
        print(
            "MISSING",
            rel,
        )

    raise RuntimeError(
        "Canonical closeout package incomplete on origin/main."
    )


print(
    "Remote file-set audit: PASS"
)


# =============================================================================
# 11. REMOTE BYTE-FOR-BYTE VERIFICATION — ALL 104 FILES
# =============================================================================

banner("STAGE26-PC1 :: REMOTE BYTE VERIFICATION — 104 FILES")


pass_count = 0


for i, rel in enumerate(
    expected_files,
    start=1,
):

    p = subprocess.run(
        [
            "git",
            "show",
            f"origin/main:{rel}",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )


    if p.returncode != 0:
        raise RuntimeError(
            f"Remote read failed:\n"
            f"{rel}\n\n"
            f"{p.stderr.decode('utf-8', errors='replace')}"
        )


    remote_sha = hashlib.sha256(
        p.stdout
    ).hexdigest()


    expected_sha = local_sha[
        rel
    ]


    if remote_sha != expected_sha:

        print()
        print(
            "BYTE MISMATCH:",
            rel,
        )

        print(
            "local :",
            expected_sha,
        )

        print(
            "remote:",
            remote_sha,
        )

        raise RuntimeError(
            f"Remote byte mismatch: {rel}"
        )


    pass_count += 1


    print(
        f"[{i:03d}/104] PASS  {rel}"
    )


print()
print(
    "Remote byte verification:",
    pass_count,
    "/ 104 PASS",
)


# =============================================================================
# 12. FINAL REPOSITORY STATE
# =============================================================================

banner("STAGE26-PC1 COMPLETE :: CANONICAL PUBLICATION CLOSEOUT DURABLE")


final_status = git(
    "status",
    "--porcelain",
)


print(
    "STAGE26 SCIENTIFIC ANCHOR :",
    STAGE26_SCIENTIFIC_ANCHOR,
)

print(
    "PUBLICATION CLOSEOUT HEAD :",
    new_head,
)

print()
print(
    "HEAD == origin/main       :",
    local_head
    ==
    remote_head
    ==
    new_head,
)

print(
    "Repository clean          :",
    final_status == "",
)

print()
print(
    "Canonical files           : 104 / 104"
)

print(
    "Canonical tables          : 12"
)

print(
    "Figure families           : 10"
)

print(
    "Publication PNGs          : 10"
)

print(
    "Publication PDFs          : 10"
)

print(
    "Top-level mirrored figures: 20 files"
)

print()
print(
    "GPU rerun                 : NO"
)

print(
    "CPU inference rerun       : NO"
)

print(
    "Timing rerun              : NO"
)

print(
    "Memory profiling rerun    : NO"
)

print(
    "Corpus access             : NO"
)

print(
    "New scientific result     : NO"
)

print()
print(
    "stage26_8 protocol folder :"
)

print(
    "  POST-CLOSURE ARCHIVAL RECONSTRUCTION / INDEX"
)

print(
    "  NOT PROSPECTIVE PREREGISTRATION"
)

print()
print(
    "STAGE26 STATUS             : COMPLETE / CLOSED"
)

print(
    "PUBLICATION PACKAGE        : COMPLETE / DURABLE"
)

print()
print(
    "NEXT:"
)

print(
    "  Paste the COMPLETE output."
)

print(
    "  If all 104 remote byte checks pass, Stage26 requires "
    "no further work and we move to the next scientific stage."
)


if final_status:
    raise RuntimeError(
        "Repository not clean after Stage26 publication closeout."
    )


STAGE26-PC1 :: PARENT / HISTORY GATE
Expected parent           : 3bf4c6e9e9cb2e6ccfeb108e35cbf5fbeded6fd4
Local HEAD                : 3bf4c6e9e9cb2e6ccfeb108e35cbf5fbeded6fd4
origin/main               : 3bf4c6e9e9cb2e6ccfeb108e35cbf5fbeded6fd4
Stage26 scientific anchor : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
Scientific anchor ancestor: True

STAGE26-PC1 :: EXACT 104-FILE PACKAGE AUDIT
Expected canonical files: 104
Current untracked files  : 104
Pre-stage SHA256 inventory: 104 / 104

STAGE26-PC1 :: PUBLICATION GEOMETRY CHECK
Canonical tables              : 12
Publication-package PNGs      : 10
Publication-package PDFs      : 10
Top-level Stage26 PNGs        : 10
Top-level Stage26 PDFs        : 10
Publication ↔ top-level figure byte identity: PASS

STAGE26-PC1 :: GPU PROTOCOL ARCHIVAL QUALIFICATION
directory_role             : POST_CLOSURE_CANONICAL_ARCHIVAL_INDEX
prospective_preregistration: False
retroactive_protocol_claim : False
new_measurement            : False

Archival qualif

In [11]:
# =============================================================================
# STAGE26-NB0 :: KAGGLE NOTEBOOK EXPORT / RECOVERY PREFLIGHT
#
# PURPOSE
#   Determine the safest source for preserving the current Stage26 Kaggle
#   notebook as a durable Python script.
#
# ORDER OF PREFERENCE
#   1. Actual current .ipynb exposed by Kaggle/Jupyter
#   2. Jupyter server/session notebook path if discoverable
#   3. Live IPython executed-cell history
#
# NO science
# NO inference
# NO timing
# NO corpus access
# NO Git modification
# NO commit/push
# =============================================================================

from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path


REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "8767f38719584ec9e8d18e5c1dfae0913e537fed"
)


def banner(text):
    print()
    print("=" * 120)
    print(text)
    print("=" * 120)


def run(cmd, *, cwd=None, check=False):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(map(str, cmd))}\n{p.stdout}"
        )

    return p.returncode, p.stdout.strip()


# =============================================================================
# 1. REPOSITORY GATE
# =============================================================================

banner("STAGE26-NB0 :: REPOSITORY GATE")

rc, head = run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    check=True,
)

rc, origin = run(
    ["git", "rev-parse", "origin/main"],
    cwd=REPO,
    check=True,
)

rc, status = run(
    ["git", "status", "--porcelain"],
    cwd=REPO,
    check=True,
)


print("Expected HEAD :", EXPECTED_HEAD)
print("Local HEAD    :", head)
print("origin/main   :", origin)
print("Repo clean    :", status == "")


if head != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected repository HEAD."
    )

if origin != EXPECTED_HEAD:
    raise RuntimeError(
        "origin/main differs from expected HEAD."
    )

if status:
    raise RuntimeError(
        "Repository must be clean."
    )


# =============================================================================
# 2. SEARCH FOR ACTUAL NOTEBOOK FILES
# =============================================================================

banner("STAGE26-NB0 :: SEARCH FOR LIVE NOTEBOOK FILES")


search_roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
    Path("/tmp"),
    Path("/root"),
]


notebook_candidates = []


for root in search_roots:

    if not root.exists():
        continue

    try:
        for p in root.rglob("*.ipynb"):

            # Avoid recursively scanning the cloned repository's known old
            # notebook as a candidate for the current live Kaggle notebook.
            try:
                rel_repo = p.relative_to(REPO)

                if str(rel_repo) == (
                    "notebooks/original_kaggle_working_notebook.ipynb"
                ):
                    continue

            except ValueError:
                pass

            try:
                stat = p.stat()

                notebook_candidates.append(
                    (
                        p,
                        stat.st_size,
                        stat.st_mtime,
                    )
                )

            except Exception:
                pass

    except Exception as exc:
        print(
            f"[WARN] Could not fully scan {root}: {exc}"
        )


notebook_candidates.sort(
    key=lambda x: x[2],
    reverse=True,
)


print(
    "Non-repository .ipynb candidates:",
    len(notebook_candidates),
)


for p, size, mtime in notebook_candidates[:30]:

    print(
        f"{size:>12,d} bytes  {p}"
    )


# =============================================================================
# 3. JUPYTER SERVER / SESSION DISCOVERY
# =============================================================================

banner("STAGE26-NB0 :: JUPYTER SERVER / SESSION DISCOVERY")


commands = [
    ["jupyter", "server", "list", "--json"],
    ["jupyter", "notebook", "list", "--json"],
]


for cmd in commands:

    rc, out = run(
        cmd
    )

    print()
    print("$", " ".join(cmd))
    print("return code:", rc)

    if out:
        print(out[:12000])
    else:
        print("<no output>")


# =============================================================================
# 4. IPYTHON LIVE HISTORY AUDIT
# =============================================================================

banner("STAGE26-NB0 :: LIVE IPYTHON HISTORY AUDIT")


try:
    ip = get_ipython()

except NameError:
    ip = None


if ip is None:
    raise RuntimeError(
        "No live IPython shell available."
    )


history = list(
    getattr(
        ip,
        "user_ns",
        {},
    ).get(
        "_ih",
        [],
    )
)


print(
    "Raw _ih entries:",
    len(history),
)


nonempty = [
    x
    for x in history
    if isinstance(x, str)
    and x.strip()
]


print(
    "Non-empty executed inputs:",
    len(nonempty),
)


total_chars = sum(
    len(x)
    for x in nonempty
)


print(
    "Total executed-input characters:",
    f"{total_chars:,}",
)


print()
print("First 5 non-empty cells:")

for i, code in enumerate(
    nonempty[:5],
    start=1,
):
    first_line = (
        code.strip()
        .splitlines()[0][:150]
    )

    print(
        f"  {i:03d}: {first_line}"
    )


print()
print("Last 15 non-empty cells:")

offset = max(
    0,
    len(nonempty) - 15,
)

for i, code in enumerate(
    nonempty[-15:],
    start=offset + 1,
):
    first_line = (
        code.strip()
        .splitlines()[0][:150]
    )

    print(
        f"  {i:03d}: {first_line}"
    )


# =============================================================================
# 5. HISTORY DATABASE AUDIT
# =============================================================================

banner("STAGE26-NB0 :: IPYTHON HISTORY DATABASE")


hm = getattr(
    ip,
    "history_manager",
    None,
)


if hm is None:
    print(
        "History manager: unavailable"
    )

else:

    hist_file = getattr(
        hm,
        "hist_file",
        None,
    )

    print(
        "History file:",
        hist_file,
    )

    if hist_file:

        hp = Path(
            str(hist_file)
        )

        print(
            "History DB exists:",
            hp.exists(),
        )

        if hp.exists():

            print(
                "History DB bytes :",
                hp.stat().st_size,
            )


# =============================================================================
# 6. CHECK WHETHER STAGE26 SCRIPT ALREADY EXISTS
# =============================================================================

banner("STAGE26-NB0 :: CURRENT REPOSITORY SCRIPT CHECK")


stage26_script_candidates = []


scripts_root = (
    REPO
    / "scripts"
)


if scripts_root.exists():

    for p in scripts_root.rglob("*"):

        if (
            p.is_file()
            and
            "stage26" in p.name.lower()
        ):

            stage26_script_candidates.append(
                p
            )


print(
    "Existing Stage26 script/notebook exports:",
    len(stage26_script_candidates),
)


for p in stage26_script_candidates:
    print(
        " ",
        p.relative_to(REPO),
    )


# =============================================================================
# 7. FINAL DECISION INPUT
# =============================================================================

banner("STAGE26-NB0 COMPLETE")


print(
    "Actual .ipynb candidates      :",
    len(notebook_candidates),
)

print(
    "Live executed code cells      :",
    len(nonempty),
)

print(
    "Live executed code characters :",
    f"{total_chars:,}",
)

print(
    "Existing Stage26 repo export  :",
    len(stage26_script_candidates),
)

print()
print(
    "Git modified                  : NO"
)

print(
    "Commit/push performed         : NO"
)

print()
print(
    "NEXT:"
)

print(
    "  Paste the COMPLETE output."
)

print(
    "  Then we will export the best available source as a Stage26 "
    "Kaggle Python script and push it to GitHub immediately."
)


STAGE26-NB0 :: REPOSITORY GATE
Expected HEAD : 8767f38719584ec9e8d18e5c1dfae0913e537fed
Local HEAD    : 8767f38719584ec9e8d18e5c1dfae0913e537fed
origin/main   : 8767f38719584ec9e8d18e5c1dfae0913e537fed
Repo clean    : True

STAGE26-NB0 :: SEARCH FOR LIVE NOTEBOOK FILES
Non-repository .ipynb candidates: 5
   3,759,782 bytes  /kaggle/working/.virtual_documents/__notebook_source__.ipynb
     905,371 bytes  /kaggle/working/ids2018-validation-safe-ablation/scripts/stage25/stage25_prevalence_operational_stress.ipynb
   2,244,097 bytes  /kaggle/working/ids2018-validation-safe-ablation/scripts/stage24/stage24_cross_dataset_generalization.ipynb
         613 bytes  /root/.julia/packages/Interpolations/dR5oF/doc/Plotting examples.ipynb
       7,087 bytes  /root/.julia/packages/Interpolations/dR5oF/doc/Interpolations.jl.ipynb

STAGE26-NB0 :: JUPYTER SERVER / SESSION DISCOVERY

$ jupyter server list --json
return code: 0
{"base_url": "/k/343685626/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2IiwidH

In [12]:
# =============================================================================
# STAGE26-NB1 :: PRESERVE LIVE KAGGLE NOTEBOOK + PYTHON SCRIPT TO GITHUB
#
# SOURCE
#   /kaggle/working/.virtual_documents/__notebook_source__.ipynb
#
# DURABLE OUTPUTS
#   notebooks/stage26_deployment_profiling_kaggle.ipynb
#   scripts/stage26/stage26_deployment_profiling_kaggle.py
#   scripts/stage26/stage26_notebook_export_receipt.json
#
# PURPOSE
#   Preserve the complete Stage26 Kaggle working notebook before the runtime
#   disappears.
#
# NO science rerun
# NO inference
# NO timing
# NO GPU
# NO corpus access
# =============================================================================

from __future__ import annotations

import hashlib
import json
import os
import shutil
import stat
import subprocess
import tempfile
from datetime import datetime, timezone
from pathlib import Path


# =============================================================================
# 0. IDENTITIES / PATHS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "8767f38719584ec9e8d18e5c1dfae0913e537fed"
)

STAGE26_SCIENTIFIC_ANCHOR = (
    "9e8354ecc9cfa72c28aa037e5d2053de422bf7a2"
)

LIVE_NOTEBOOK = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)

NOTEBOOK_DST = (
    REPO
    / "notebooks"
    / "stage26_deployment_profiling_kaggle.ipynb"
)

SCRIPT_DIR = (
    REPO
    / "scripts"
    / "stage26"
)

SCRIPT_DST = (
    SCRIPT_DIR
    / "stage26_deployment_profiling_kaggle.py"
)

RECEIPT_DST = (
    SCRIPT_DIR
    / "stage26_notebook_export_receipt.json"
)

COMMIT_MSG = (
    "stage26: preserve Kaggle notebook and script"
)


# =============================================================================
# HELPERS
# =============================================================================

def banner(text):
    print()
    print("=" * 122)
    print(text)
    print("=" * 122)


def run(cmd, *, cwd=REPO, env=None, check=True):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(map(str, cmd))}\n\n"
            f"{p.stdout}"
        )

    return p.stdout.strip()


def git(*args, env=None, check=True):
    return run(
        ["git", *args],
        env=env,
        check=check,
    )


def sha256_bytes(data):
    return hashlib.sha256(
        data
    ).hexdigest()


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for block in iter(
            lambda: f.read(16 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def repo_rel(path):
    return str(
        Path(path).relative_to(REPO)
    )


# =============================================================================
# 1. REPOSITORY GATE
# =============================================================================

banner("STAGE26-NB1 :: REPOSITORY GATE")

git(
    "fetch",
    "origin",
    "main",
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print("Expected parent           :", EXPECTED_PARENT)
print("Local HEAD                :", head)
print("origin/main               :", origin)
print("Repository clean          :", status == "")
print("Stage26 scientific anchor :", STAGE26_SCIENTIFIC_ANCHOR)


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Unexpected local HEAD."
    )

if origin != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main differs from expected parent."
    )

if status:
    raise RuntimeError(
        "Repository must be clean."
    )


rc = subprocess.run(
    [
        "git",
        "merge-base",
        "--is-ancestor",
        STAGE26_SCIENTIFIC_ANCHOR,
        head,
    ],
    cwd=REPO,
    check=False,
).returncode


print(
    "Scientific anchor ancestor:",
    rc == 0,
)


if rc != 0:
    raise RuntimeError(
        "Stage26 scientific anchor missing from history."
    )


# =============================================================================
# 2. LIVE NOTEBOOK SNAPSHOT
# =============================================================================

banner("STAGE26-NB1 :: SNAPSHOT LIVE KAGGLE NOTEBOOK")

if not LIVE_NOTEBOOK.is_file():
    raise FileNotFoundError(
        LIVE_NOTEBOOK
    )


# Read ONCE so notebook and script originate from exactly one immutable snapshot.
notebook_bytes = LIVE_NOTEBOOK.read_bytes()

notebook_sha = sha256_bytes(
    notebook_bytes
)


print(
    "Live notebook:",
    LIVE_NOTEBOOK,
)

print(
    "Snapshot bytes:",
    f"{len(notebook_bytes):,}",
)

print(
    "Snapshot SHA256:",
    notebook_sha,
)


if len(notebook_bytes) < 1_000_000:
    raise RuntimeError(
        "Live notebook snapshot is unexpectedly small."
    )


# =============================================================================
# 3. PARSE / AUDIT NOTEBOOK
# =============================================================================

banner("STAGE26-NB1 :: NOTEBOOK STRUCTURE AUDIT")

try:
    notebook_obj = json.loads(
        notebook_bytes.decode(
            "utf-8"
        )
    )

except Exception as exc:
    raise RuntimeError(
        f"Unable to parse live notebook JSON: {exc}"
    )


if notebook_obj.get(
    "nbformat"
) is None:
    raise RuntimeError(
        "Source is not a valid Jupyter notebook."
    )


cells = notebook_obj.get(
    "cells",
    []
)

code_cells = [
    c
    for c in cells
    if c.get("cell_type") == "code"
]

markdown_cells = [
    c
    for c in cells
    if c.get("cell_type") == "markdown"
]


output_cells = [
    c
    for c in code_cells
    if c.get("outputs")
]


output_objects = sum(
    len(
        c.get(
            "outputs",
            []
        )
    )
    for c in code_cells
)


code_chars = sum(
    len(
        c.get(
            "source",
            ""
        )
        if isinstance(
            c.get(
                "source",
                ""
            ),
            str,
        )
        else "".join(
            c.get(
                "source",
                []
            )
        )
    )
    for c in code_cells
)


print(
    "nbformat              :",
    notebook_obj.get(
        "nbformat"
    ),
    ".",
    notebook_obj.get(
        "nbformat_minor"
    ),
    sep="",
)

print(
    "Total cells           :",
    len(cells),
)

print(
    "Code cells            :",
    len(code_cells),
)

print(
    "Markdown cells        :",
    len(markdown_cells),
)

print(
    "Cells containing output:",
    len(output_cells),
)

print(
    "Output objects        :",
    output_objects,
)

print(
    "Code-source characters:",
    f"{code_chars:,}",
)


if len(code_cells) < 10:
    raise RuntimeError(
        "Live notebook contains unexpectedly few code cells."
    )


# =============================================================================
# 4. PRESERVE EXACT .IPYNB SNAPSHOT
# =============================================================================

banner("STAGE26-NB1 :: PRESERVE EXACT NOTEBOOK")

if NOTEBOOK_DST.exists():
    raise RuntimeError(
        f"Destination already exists: {NOTEBOOK_DST}"
    )

if SCRIPT_DST.exists():
    raise RuntimeError(
        f"Destination already exists: {SCRIPT_DST}"
    )

if RECEIPT_DST.exists():
    raise RuntimeError(
        f"Destination already exists: {RECEIPT_DST}"
    )


NOTEBOOK_DST.parent.mkdir(
    parents=True,
    exist_ok=True,
)

SCRIPT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


NOTEBOOK_DST.write_bytes(
    notebook_bytes
)


notebook_copy_sha = sha256_file(
    NOTEBOOK_DST
)


print(
    "Repository notebook:",
    repo_rel(
        NOTEBOOK_DST
    ),
)

print(
    "Copied bytes       :",
    f"{NOTEBOOK_DST.stat().st_size:,}",
)

print(
    "Copied SHA256      :",
    notebook_copy_sha,
)

print(
    "Byte-identical     :",
    notebook_copy_sha == notebook_sha,
)


if notebook_copy_sha != notebook_sha:
    raise RuntimeError(
        "Notebook copy is not byte-identical."
    )


# =============================================================================
# 5. DETERMINISTIC PYTHON SCRIPT EXPORT
# =============================================================================

banner("STAGE26-NB1 :: EXPORT NOTEBOOK AS PYTHON SCRIPT")


def cell_source(cell):
    src = cell.get(
        "source",
        ""
    )

    if isinstance(
        src,
        list,
    ):
        return "".join(
            src
        )

    return str(
        src
    )


script_parts = []

script_parts.append(
    '''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Stage26 Kaggle deployment-profiling notebook export.

This file is a deterministic source export of the live Kaggle notebook
preserved at:

    notebooks/stage26_deployment_profiling_kaggle.ipynb

Scientific Stage26 anchor:
    9e8354ecc9cfa72c28aa037e5d2053de422bf7a2

IMPORTANT
---------
This script preserves notebook source for reproducibility and archival use.
The accompanying .ipynb is the authoritative exact snapshot and retains
notebook outputs/metadata.

Do not interpret execution of this file as required to reproduce already-frozen
Stage26 publication results. Stage26 science is complete and closed.
"""
'''
)


for idx, cell in enumerate(
    cells,
    start=1,
):

    ctype = cell.get(
        "cell_type",
        "unknown"
    )

    src = cell_source(
        cell
    )


    if ctype == "markdown":

        script_parts.append(
            f"\n# %% [markdown] cell {idx}\n"
        )

        for line in src.splitlines():

            if line:
                script_parts.append(
                    "# " + line + "\n"
                )
            else:
                script_parts.append(
                    "#\n"
                )


    elif ctype == "code":

        script_parts.append(
            f"\n# %% cell {idx}\n"
        )

        script_parts.append(
            src
        )

        if src and not src.endswith(
            "\n"
        ):
            script_parts.append(
                "\n"
            )


    elif ctype == "raw":

        script_parts.append(
            f"\n# %% [raw] cell {idx}\n"
        )

        for line in src.splitlines():

            script_parts.append(
                "# RAW: " + line + "\n"
            )


    else:

        script_parts.append(
            f"\n# %% [unknown:{ctype}] cell {idx}\n"
        )

        for line in src.splitlines():

            script_parts.append(
                "# " + line + "\n"
            )


script_text = "".join(
    script_parts
)


SCRIPT_DST.write_text(
    script_text,
    encoding="utf-8",
)


script_sha = sha256_file(
    SCRIPT_DST
)


print(
    "Python script      :",
    repo_rel(
        SCRIPT_DST
    ),
)

print(
    "Script bytes       :",
    f"{SCRIPT_DST.stat().st_size:,}",
)

print(
    "Script SHA256      :",
    script_sha,
)

print(
    "Exported cells     :",
    len(cells),
)


# =============================================================================
# 6. SCRIPT CONTENT SANITY CHECK
# =============================================================================

banner("STAGE26-NB1 :: SCRIPT SANITY CHECK")


required_markers = [
    "STAGE26",
    "Stage26",
    "stage26",
]


for marker in required_markers:

    present = (
        marker
        in script_text
    )

    print(
        f"{marker:<10s}:",
        present,
    )


if "stage26" not in script_text.lower():
    raise RuntimeError(
        "Exported script does not appear to contain Stage26 source."
    )


print()
print(
    "Python export contains:",
    f"{script_text.count('# %% cell ')} code-cell markers",
)

print(
    "Markdown export contains:",
    f"{script_text.count('# %% [markdown]')} markdown-cell markers",
)


# =============================================================================
# 7. CREATE EXPORT RECEIPT
# =============================================================================

banner("STAGE26-NB1 :: CREATE EXPORT RECEIPT")


receipt = {
    "schema":
        "stage26_kaggle_notebook_export_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "source_live_notebook":
        str(
            LIVE_NOTEBOOK
        ),

    "scientific_stage26_anchor":
        STAGE26_SCIENTIFIC_ANCHOR,

    "publication_closeout_parent":
        EXPECTED_PARENT,

    "notebook": {
        "repository_path":
            repo_rel(
                NOTEBOOK_DST
            ),

        "bytes":
            NOTEBOOK_DST.stat().st_size,

        "sha256":
            notebook_copy_sha,

        "exact_live_snapshot":
            True,

        "nbformat":
            notebook_obj.get(
                "nbformat"
            ),

        "nbformat_minor":
            notebook_obj.get(
                "nbformat_minor"
            ),

        "total_cells":
            len(cells),

        "code_cells":
            len(code_cells),

        "markdown_cells":
            len(markdown_cells),

        "cells_with_outputs":
            len(output_cells),

        "output_objects":
            output_objects,
    },

    "script": {
        "repository_path":
            repo_rel(
                SCRIPT_DST
            ),

        "bytes":
            SCRIPT_DST.stat().st_size,

        "sha256":
            script_sha,

        "export_type":
            "DETERMINISTIC_CELL_SOURCE_EXPORT",

        "notebook_outputs_embedded":
            False,
    },

    "science": {
        "new_measurement":
            False,

        "gpu_used":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "memory_profiling_performed":
            False,

        "corpus_accessed":
            False,

        "model_loaded":
            False,
    },
}


RECEIPT_DST.write_text(
    json.dumps(
        receipt,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


receipt_sha = sha256_file(
    RECEIPT_DST
)


print(
    "Receipt            :",
    repo_rel(
        RECEIPT_DST
    ),
)

print(
    "Receipt SHA256     :",
    receipt_sha,
)


# =============================================================================
# 8. EXACT LOCAL FILE-SET AUDIT
# =============================================================================

banner("STAGE26-NB1 :: EXACT LOCAL FILE-SET AUDIT")


expected_files = sorted(
    [
        repo_rel(
            NOTEBOOK_DST
        ),
        repo_rel(
            SCRIPT_DST
        ),
        repo_rel(
            RECEIPT_DST
        ),
    ]
)


untracked = sorted(
    x
    for x in git(
        "ls-files",
        "--others",
        "--exclude-standard",
    ).splitlines()
    if x
)


tracked_changes = git(
    "diff",
    "--name-only",
)


print(
    "Expected new files:",
    len(expected_files),
)

for rel in expected_files:
    print(
        " ",
        rel
    )


print()
print(
    "Actual untracked:",
    len(untracked),
)


if untracked != expected_files:

    print(
        "Actual untracked set:"
    )

    for rel in untracked:
        print(
            " ",
            rel
        )

    raise RuntimeError(
        "Unexpected untracked file set."
    )


if tracked_changes:
    raise RuntimeError(
        "Unexpected tracked modifications:\n"
        + tracked_changes
    )


local_sha = {
    repo_rel(
        NOTEBOOK_DST
    ):
        notebook_copy_sha,

    repo_rel(
        SCRIPT_DST
    ):
        script_sha,

    repo_rel(
        RECEIPT_DST
    ):
        receipt_sha,
}


# =============================================================================
# 9. STAGE EXACTLY THREE FILES
# =============================================================================

banner("STAGE26-NB1 :: STAGE EXACT EXPORT PACKAGE")


git(
    "add",
    "--",
    *expected_files,
)


staged = sorted(
    x
    for x in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if x
)


print(
    "Expected staged:",
    len(expected_files),
)

print(
    "Actual staged  :",
    len(staged),
)


for rel in staged:
    print(
        " ",
        rel
    )


if staged != expected_files:
    raise RuntimeError(
        "Staged set differs from expected notebook export package."
    )


if git(
    "diff",
    "--name-only",
):
    raise RuntimeError(
        "Unexpected unstaged tracked changes."
    )


if git(
    "ls-files",
    "--others",
    "--exclude-standard",
):
    raise RuntimeError(
        "Unexpected untracked files remain."
    )


# =============================================================================
# 10. COMMIT
# =============================================================================

banner("STAGE26-NB1 :: COMMIT")


git(
    "commit",
    "-m",
    COMMIT_MSG,
)


new_head = git(
    "rev-parse",
    "HEAD",
)

parent = git(
    "rev-parse",
    "HEAD^",
)

subject = git(
    "show",
    "-s",
    "--format=%s",
    new_head,
)


print(
    "Parent  :",
    parent,
)

print(
    "New HEAD:",
    new_head,
)

print(
    "Subject :",
    subject,
)


if parent != EXPECTED_PARENT:
    raise RuntimeError(
        "Notebook export commit has unexpected parent."
    )


if subject != COMMIT_MSG:
    raise RuntimeError(
        "Unexpected commit subject."
    )


# =============================================================================
# 11. PUSH SECURELY
# =============================================================================

banner("STAGE26-NB1 :: PUSH")


from kaggle_secrets import UserSecretsClient


token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not token:
    raise RuntimeError(
        "GITHUB_TOKEN unavailable."
    )


fd, askpass_name = tempfile.mkstemp(
    prefix="stage26_nb1_askpass_",
    suffix=".sh",
)

os.close(
    fd
)

askpass = Path(
    askpass_name
)


try:

    askpass.write_text(
        "#!/bin/sh\n"
        "case \"$1\" in\n"
        "  *Username*) printf '%s\\n' 'x-access-token' ;;\n"
        "  *Password*) printf '%s\\n' \"$TOKEN_VALUE\" ;;\n"
        "  *) printf '%s\\n' '' ;;\n"
        "esac\n",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        | stat.S_IXUSR
    )


    env = os.environ.copy()

    env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    env[
        "TOKEN_VALUE"
    ] = token


    print(
        git(
            "push",
            "origin",
            "main",
            env=env,
        )
    )


finally:

    askpass.unlink(
        missing_ok=True
    )

    token = None


# =============================================================================
# 12. REMOTE HEAD VERIFICATION
# =============================================================================

banner("STAGE26-NB1 :: REMOTE HEAD VERIFICATION")


git(
    "fetch",
    "origin",
    "main",
)


local_head = git(
    "rev-parse",
    "HEAD",
)

remote_head = git(
    "rev-parse",
    "origin/main",
)


print(
    "Local HEAD :",
    local_head,
)

print(
    "origin/main:",
    remote_head,
)


if not (
    local_head
    ==
    remote_head
    ==
    new_head
):
    raise RuntimeError(
        "Local/remote HEAD mismatch."
    )


# =============================================================================
# 13. REMOTE BYTE VERIFICATION
# =============================================================================

banner("STAGE26-NB1 :: REMOTE BYTE VERIFICATION")


for rel in expected_files:

    p = subprocess.run(
        [
            "git",
            "show",
            f"origin/main:{rel}",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )


    if p.returncode != 0:
        raise RuntimeError(
            f"Unable to read remote file: {rel}"
        )


    remote_sha = hashlib.sha256(
        p.stdout
    ).hexdigest()


    expected_sha = local_sha[
        rel
    ]


    print(
        rel
    )

    print(
        "  local :",
        expected_sha
    )

    print(
        "  remote:",
        remote_sha
    )

    print(
        "  PASS  :",
        expected_sha
        ==
        remote_sha
    )


    if remote_sha != expected_sha:
        raise RuntimeError(
            f"Remote byte mismatch: {rel}"
        )


# =============================================================================
# 14. FINAL STATE
# =============================================================================

banner("STAGE26-NB1 COMPLETE :: NOTEBOOK DURABLY PRESERVED")


final_status = git(
    "status",
    "--porcelain",
)


print(
    "Scientific Stage26 anchor :",
    STAGE26_SCIENTIFIC_ANCHOR,
)

print(
    "Notebook preservation HEAD:",
    new_head,
)

print()
print(
    "Exact notebook snapshot    :",
    repo_rel(
        NOTEBOOK_DST
    ),
)

print(
    "Notebook SHA256            :",
    notebook_copy_sha,
)

print()
print(
    "Python script export       :",
    repo_rel(
        SCRIPT_DST
    ),
)

print(
    "Script SHA256              :",
    script_sha,
)

print()
print(
    "Export receipt             :",
    repo_rel(
        RECEIPT_DST
    ),
)

print(
    "Receipt SHA256             :",
    receipt_sha,
)

print()
print(
    "HEAD == origin/main        :",
    local_head
    ==
    remote_head
    ==
    new_head,
)

print(
    "Repository clean           :",
    final_status == "",
)

print()
print(
    "Live Kaggle notebook saved : YES"
)

print(
    "Python source saved        : YES"
)

print(
    "Notebook outputs preserved : YES — in exact .ipynb snapshot"
)

print(
    "New Stage26 science        : NO"
)

print(
    "GPU used                   : NO"
)

print()
print(
    "STAGE26 ARCHIVAL STATE     : COMPLETE"
)


if final_status:
    raise RuntimeError(
        "Repository not clean after notebook preservation."
    )


STAGE26-NB1 :: REPOSITORY GATE
Expected parent           : 8767f38719584ec9e8d18e5c1dfae0913e537fed
Local HEAD                : 8767f38719584ec9e8d18e5c1dfae0913e537fed
origin/main               : 8767f38719584ec9e8d18e5c1dfae0913e537fed
Repository clean          : True
Stage26 scientific anchor : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2
Scientific anchor ancestor: True

STAGE26-NB1 :: SNAPSHOT LIVE KAGGLE NOTEBOOK
Live notebook: /kaggle/working/.virtual_documents/__notebook_source__.ipynb
Snapshot bytes: 3,782,162
Snapshot SHA256: a6b39be7aef8d56bb3c145dbd071bbf967c96f6f701935c4fa6c0ed389ac53f4

STAGE26-NB1 :: NOTEBOOK STRUCTURE AUDIT


RuntimeError: Unable to parse live notebook JSON: Expecting value: line 1 column 1 (char 0)

In [13]:
# =============================================================================
# STAGE26-NB1-R1 :: IDENTIFY KAGGLE VIRTUAL NOTEBOOK SOURCE FORMAT
#
# Read-only diagnostic.
# NO repository modification.
# NO commit/push.
# =============================================================================

from pathlib import Path
import hashlib
import subprocess

SRC = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "8767f38719584ec9e8d18e5c1dfae0913e537fed"
)


def banner(text):
    print()
    print("=" * 120)
    print(text)
    print("=" * 120)


def sha256_file(path):
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(16 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


# =============================================================================
# 1. REPOSITORY SAFETY CHECK
# =============================================================================

banner("NB1-R1 :: REPOSITORY SAFETY CHECK")

head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    text=True,
).strip()

status = subprocess.check_output(
    ["git", "status", "--porcelain"],
    cwd=REPO,
    text=True,
).strip()

print("Expected HEAD :", EXPECTED_HEAD)
print("Actual HEAD   :", head)
print("Repo clean    :", status == "")

if head != EXPECTED_HEAD:
    raise RuntimeError("Unexpected repository HEAD.")

if status:
    raise RuntimeError(
        "Repository is not clean after failed NB1 attempt:\n"
        + status
    )


# =============================================================================
# 2. BASIC FILE IDENTITY
# =============================================================================

banner("NB1-R1 :: SOURCE FILE IDENTITY")

if not SRC.is_file():
    raise FileNotFoundError(SRC)

print("Path   :", SRC)
print("Bytes  :", f"{SRC.stat().st_size:,}")
print("SHA256 :", sha256_file(SRC))

p = subprocess.run(
    ["file", "-b", str(SRC)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)

print("file(1):", p.stdout.strip())


# =============================================================================
# 3. FIRST RAW BYTES
# =============================================================================

banner("NB1-R1 :: FIRST RAW BYTES")

raw = SRC.read_bytes()

print("First 64 bytes repr:")
print(repr(raw[:64]))

print()
print("Starts with JSON object '{' :", raw.lstrip().startswith(b"{"))
print("Starts with JSON array '['  :", raw.lstrip().startswith(b"["))
print("Contains NUL bytes          :", b"\x00" in raw[:10000])


# =============================================================================
# 4. TEXT DECODING
# =============================================================================

banner("NB1-R1 :: TEXT DECODING")

for encoding in [
    "utf-8",
    "utf-8-sig",
    "latin-1",
]:

    try:
        text = raw.decode(encoding)

        print(
            f"{encoding:<10s}: PASS "
            f"({len(text):,} characters)"
        )

        if encoding in {
            "utf-8",
            "utf-8-sig",
        }:
            decoded = text
            break

    except Exception as exc:

        print(
            f"{encoding:<10s}: FAIL "
            f"{type(exc).__name__}: {exc}"
        )

else:
    decoded = None


if decoded is None:
    raise RuntimeError(
        "Unable to decode virtual notebook source as text."
    )


# =============================================================================
# 5. SHOW FIRST / LAST SOURCE LINES
# =============================================================================

banner("NB1-R1 :: SOURCE PREVIEW")

lines = decoded.splitlines()

print("Total lines:", f"{len(lines):,}")

print()
print("FIRST 40 LINES")
print("-" * 120)

for i, line in enumerate(
    lines[:40],
    start=1,
):
    print(
        f"{i:05d}: {line[:240]}"
    )


print()
print("LAST 40 LINES")
print("-" * 120)

start = max(
    1,
    len(lines) - 39,
)

for i, line in enumerate(
    lines[-40:],
    start=start,
):
    print(
        f"{i:05d}: {line[:240]}"
    )


# =============================================================================
# 6. FORMAT MARKER COUNTS
# =============================================================================

banner("NB1-R1 :: FORMAT MARKER COUNTS")

markers = {
    "# %%":
        decoded.count("# %%"),

    "# In[":
        decoded.count("# In["),

    "\"cells\"":
        decoded.count('"cells"'),

    "\"cell_type\"":
        decoded.count('"cell_type"'),

    "# ===":
        decoded.count("# ==="),

    "STAGE26":
        decoded.upper().count("STAGE26"),

    "get_ipython":
        decoded.count("get_ipython"),

    "UserSecretsClient":
        decoded.count("UserSecretsClient"),
}


for name, count in markers.items():
    print(
        f"{name:<20s}: {count:,}"
    )


# =============================================================================
# 7. PYTHON COMPILATION PROBE
# =============================================================================

banner("NB1-R1 :: PYTHON SOURCE PROBE")

try:

    compile(
        decoded,
        str(SRC),
        "exec",
    )

    python_compiles = True
    compile_error = None

except Exception as exc:

    python_compiles = False
    compile_error = (
        f"{type(exc).__name__}: {exc}"
    )


print(
    "Compiles as Python:",
    python_compiles,
)

if compile_error:
    print(
        "Compile diagnostic:",
        compile_error,
    )


# =============================================================================
# 8. FINAL DIAGNOSIS
# =============================================================================

banner("STAGE26-NB1-R1 COMPLETE")

looks_json = (
    raw.lstrip().startswith(b"{")
    and '"cells"' in decoded[:100000]
)

looks_python = (
    python_compiles
    or
    decoded.count("# %%") > 0
    or
    decoded.count("# In[") > 0
)


print("Looks like real ipynb JSON :", looks_json)
print("Looks like Python source   :", looks_python)

print()
print("Repository modified        : NO")
print("Commit/push performed      : NO")

print()
print("NEXT:")
print(
    "  Paste the COMPLETE output. "
    "If this is Kaggle's Python virtual source, "
    "we will preserve it byte-for-byte as the Stage26 script "
    "instead of incorrectly parsing it as notebook JSON."
)


NB1-R1 :: REPOSITORY SAFETY CHECK
Expected HEAD : 8767f38719584ec9e8d18e5c1dfae0913e537fed
Actual HEAD   : 8767f38719584ec9e8d18e5c1dfae0913e537fed
Repo clean    : True

NB1-R1 :: SOURCE FILE IDENTITY
Path   : /kaggle/working/.virtual_documents/__notebook_source__.ipynb
Bytes  : 3,788,688
SHA256 : 72a5aded351a774c4694f938e22abeacf18fa30820b1403c25472f88d966c91c
file(1): Python script, Unicode text, UTF-8 text executable

NB1-R1 :: FIRST RAW BYTES
First 64 bytes repr:
b'# This Python 3 environment comes with many helpful analytics li'

Starts with JSON object '{' : False
Starts with JSON array '['  : False
Contains NUL bytes          : False

NB1-R1 :: TEXT DECODING
utf-8     : PASS (3,788,331 characters)

NB1-R1 :: SOURCE PREVIEW
Total lines: 209,399

FIRST 40 LINES
------------------------------------------------------------------------------------------------------------------------
00001: # This Python 3 environment comes with many helpful analytics libraries installed
00002: # It 

In [14]:
# =============================================================================
# STAGE26-NB2 :: PRESERVE EXACT KAGGLE VIRTUAL NOTEBOOK SOURCE AS SCRIPT
#
# SOURCE:
#   /kaggle/working/.virtual_documents/__notebook_source__.ipynb
#
# IMPORTANT:
#   Despite its .ipynb filename, Kaggle exposes this as UTF-8 Python source.
#   We preserve it BYTE-FOR-BYTE without parsing, compiling, rewriting,
#   normalizing, or "fixing" it.
#
# DURABLE OUTPUTS:
#   scripts/stage26/stage26_deployment_profiling_kaggle.py
#   scripts/stage26/stage26_notebook_export_receipt.json
#
# NO SCIENCE
# NO GPU
# NO INFERENCE
# NO TIMING
# NO CORPUS
# =============================================================================

from __future__ import annotations

import hashlib
import json
import os
import stat
import subprocess
import tempfile
from datetime import datetime, timezone
from pathlib import Path


# =============================================================================
# 0. IDENTITIES / PATHS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "8767f38719584ec9e8d18e5c1dfae0913e537fed"
)

STAGE26_SCIENTIFIC_ANCHOR = (
    "9e8354ecc9cfa72c28aa037e5d2053de422bf7a2"
)

SRC = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)

SCRIPT_DIR = (
    REPO
    / "scripts"
    / "stage26"
)

SCRIPT_DST = (
    SCRIPT_DIR
    / "stage26_deployment_profiling_kaggle.py"
)

RECEIPT_DST = (
    SCRIPT_DIR
    / "stage26_notebook_export_receipt.json"
)

COMMIT_MSG = (
    "stage26: preserve Kaggle notebook source script"
)


# =============================================================================
# HELPERS
# =============================================================================

def banner(text):
    print()
    print("=" * 122)
    print(text)
    print("=" * 122)


def run(cmd, *, cwd=REPO, env=None, check=True):
    p = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(map(str, cmd))}\n\n{p.stdout}"
        )

    return p.stdout.strip()


def git(*args, env=None, check=True):
    return run(
        ["git", *args],
        env=env,
        check=check,
    )


def sha256_bytes(data):
    return hashlib.sha256(
        data
    ).hexdigest()


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for block in iter(
            lambda: f.read(16 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def repo_rel(path):
    return str(
        Path(path).relative_to(REPO)
    )


# =============================================================================
# 1. REPOSITORY GATE
# =============================================================================

banner("STAGE26-NB2 :: REPOSITORY GATE")

git(
    "fetch",
    "origin",
    "main",
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)


print("Expected parent           :", EXPECTED_PARENT)
print("Local HEAD                :", head)
print("origin/main               :", origin)
print("Repository clean          :", status == "")
print("Stage26 scientific anchor :", STAGE26_SCIENTIFIC_ANCHOR)


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Unexpected repository HEAD."
    )

if origin != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main differs from expected parent."
    )

if status:
    raise RuntimeError(
        "Repository must be clean."
    )


# =============================================================================
# 2. SOURCE SNAPSHOT
# =============================================================================

banner("STAGE26-NB2 :: SNAPSHOT KAGGLE VIRTUAL SOURCE")

if not SRC.is_file():
    raise FileNotFoundError(
        SRC
    )


# Read once. Everything below uses this exact immutable snapshot.
source_bytes = SRC.read_bytes()

source_sha = sha256_bytes(
    source_bytes
)


print("Source path   :", SRC)
print("Source bytes  :", f"{len(source_bytes):,}")
print("Source SHA256 :", source_sha)


if len(source_bytes) < 3_000_000:
    raise RuntimeError(
        "Virtual notebook source is unexpectedly small."
    )


# Must be UTF-8 text.
try:
    source_text = source_bytes.decode(
        "utf-8"
    )

except UnicodeDecodeError as exc:
    raise RuntimeError(
        f"Virtual notebook source is not UTF-8: {exc}"
    )


print(
    "UTF-8 characters:",
    f"{len(source_text):,}",
)

print(
    "Source lines     :",
    f"{len(source_text.splitlines()):,}",
)

print(
    "STAGE26 mentions :",
    f"{source_text.upper().count('STAGE26'):,}",
)


if "STAGE 26" not in source_text.upper() and "STAGE26" not in source_text.upper():
    raise RuntimeError(
        "Source does not appear to contain Stage26 work."
    )


# =============================================================================
# 3. VERIFY THIS IS KAGGLE VIRTUAL PYTHON SOURCE, NOT JSON
# =============================================================================

banner("STAGE26-NB2 :: SOURCE FORMAT QUALIFICATION")


starts_json_object = (
    source_bytes.lstrip().startswith(
        b"{"
    )
)

starts_json_array = (
    source_bytes.lstrip().startswith(
        b"["
    )
)


print(
    "Starts as JSON object:",
    starts_json_object,
)

print(
    "Starts as JSON array :",
    starts_json_array,
)

print(
    "First line           :",
    source_text.splitlines()[0][:200],
)


if starts_json_object or starts_json_array:
    raise RuntimeError(
        "Source format unexpectedly changed to JSON."
    )


# Standalone compilation is informative only.
# FAILURE DOES NOT BLOCK archival preservation.
try:

    compile(
        source_text,
        str(SRC),
        "exec",
    )

    standalone_python_compiles = True
    compile_diagnostic = None

except Exception as exc:

    standalone_python_compiles = False

    compile_diagnostic = (
        f"{type(exc).__name__}: {exc}"
    )


print(
    "Standalone compile:",
    standalone_python_compiles,
)

if compile_diagnostic:

    print(
        "Compile diagnostic:",
        compile_diagnostic,
    )

print()
print(
    "Compilation is NOT an archival acceptance criterion."
)


# =============================================================================
# 4. DESTINATION GATE
# =============================================================================

banner("STAGE26-NB2 :: DESTINATION GATE")


if SCRIPT_DST.exists():
    raise RuntimeError(
        f"Stage26 script already exists: {SCRIPT_DST}"
    )

if RECEIPT_DST.exists():
    raise RuntimeError(
        f"Export receipt already exists: {RECEIPT_DST}"
    )


SCRIPT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# 5. BYTE-FOR-BYTE SCRIPT PRESERVATION
# =============================================================================

banner("STAGE26-NB2 :: BYTE-FOR-BYTE SCRIPT PRESERVATION")


SCRIPT_DST.write_bytes(
    source_bytes
)


script_sha = sha256_file(
    SCRIPT_DST
)


print(
    "Destination:",
    repo_rel(
        SCRIPT_DST
    ),
)

print(
    "Source SHA :",
    source_sha,
)

print(
    "Script SHA :",
    script_sha,
)

print(
    "Identical  :",
    source_sha == script_sha,
)


if source_sha != script_sha:
    raise RuntimeError(
        "Byte-for-byte Stage26 script preservation failed."
    )


# =============================================================================
# 6. CREATE EXPORT RECEIPT
# =============================================================================

banner("STAGE26-NB2 :: CREATE EXPORT RECEIPT")


receipt = {
    "schema":
        "stage26_kaggle_virtual_source_export_receipt_v1",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_stage26_anchor":
        STAGE26_SCIENTIFIC_ANCHOR,

    "repository_parent_before_export":
        EXPECTED_PARENT,

    "source": {
        "path":
            str(
                SRC
            ),

        "observed_type":
            "KAGGLE_VIRTUAL_PYTHON_SOURCE",

        "misleading_filename_extension":
            ".ipynb",

        "actual_json_notebook":
            False,

        "utf8_decodable":
            True,

        "bytes":
            len(
                source_bytes
            ),

        "characters":
            len(
                source_text
            ),

        "lines":
            len(
                source_text.splitlines()
            ),

        "sha256":
            source_sha,

        "standalone_python_compiles":
            standalone_python_compiles,

        "compile_diagnostic":
            compile_diagnostic,
    },

    "destination": {
        "repository_path":
            repo_rel(
                SCRIPT_DST
            ),

        "bytes":
            SCRIPT_DST.stat().st_size,

        "sha256":
            script_sha,

        "preservation_mode":
            "BYTE_FOR_BYTE",

        "rewritten":
            False,

        "normalized":
            False,

        "syntax_fixed":
            False,

        "cells_reconstructed":
            False,
    },

    "interpretation": {
        "authoritative_role":
            "ARCHIVAL_KAGGLE_NOTEBOOK_SOURCE_SCRIPT",

        "standalone_executability_guaranteed":
            False,

        "reason":
            (
                "Kaggle virtual notebook source can concatenate notebook "
                "constructs, magics, and cell fragments that are not guaranteed "
                "to compile as one standalone Python module."
            ),

        "scientific_results_source":
            (
                "Frozen Stage26 repository artifacts and receipts remain the "
                "authoritative scientific results."
            ),
    },

    "science": {
        "new_measurement":
            False,

        "gpu_used":
            False,

        "inference_performed":
            False,

        "timing_performed":
            False,

        "memory_profiling_performed":
            False,

        "model_loaded":
            False,

        "corpus_accessed":
            False,
    },
}


RECEIPT_DST.write_text(
    json.dumps(
        receipt,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


receipt_sha = sha256_file(
    RECEIPT_DST
)


print(
    "Receipt:",
    repo_rel(
        RECEIPT_DST
    ),
)

print(
    "Receipt SHA256:",
    receipt_sha,
)


# =============================================================================
# 7. EXACT LOCAL FILE-SET AUDIT
# =============================================================================

banner("STAGE26-NB2 :: EXACT LOCAL FILE-SET AUDIT")


expected_files = sorted(
    [
        repo_rel(
            SCRIPT_DST
        ),
        repo_rel(
            RECEIPT_DST
        ),
    ]
)


untracked = sorted(
    x
    for x in git(
        "ls-files",
        "--others",
        "--exclude-standard",
    ).splitlines()
    if x
)


tracked_diff = git(
    "diff",
    "--name-only",
)


print(
    "Expected new files:",
    len(expected_files),
)

for rel in expected_files:
    print(
        " ",
        rel
    )


print(
    "Actual untracked  :",
    len(untracked),
)


if untracked != expected_files:

    print()
    print(
        "Actual untracked files:"
    )

    for rel in untracked:
        print(
            " ",
            rel
        )

    raise RuntimeError(
        "Unexpected local untracked file set."
    )


if tracked_diff:
    raise RuntimeError(
        "Unexpected tracked modifications:\n"
        + tracked_diff
    )


local_sha = {
    repo_rel(
        SCRIPT_DST
    ):
        script_sha,

    repo_rel(
        RECEIPT_DST
    ):
        receipt_sha,
}


# =============================================================================
# 8. STAGE EXACTLY TWO FILES
# =============================================================================

banner("STAGE26-NB2 :: STAGE EXACT EXPORT PACKAGE")


git(
    "add",
    "--",
    *expected_files,
)


staged = sorted(
    x
    for x in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if x
)


print(
    "Expected staged:",
    len(expected_files),
)

print(
    "Actual staged  :",
    len(staged),
)


for rel in staged:
    print(
        " ",
        rel
    )


if staged != expected_files:
    raise RuntimeError(
        "Unexpected staged file set."
    )


if git(
    "diff",
    "--name-only",
):
    raise RuntimeError(
        "Unexpected unstaged tracked changes."
    )


if git(
    "ls-files",
    "--others",
    "--exclude-standard",
):
    raise RuntimeError(
        "Unexpected untracked files remain."
    )


# =============================================================================
# 9. PRE-COMMIT BYTE AUDIT
# =============================================================================

banner("STAGE26-NB2 :: PRE-COMMIT BYTE AUDIT")


if sha256_file(
    SCRIPT_DST
) != source_sha:

    raise RuntimeError(
        "Stage26 script changed before commit."
    )


if sha256_file(
    RECEIPT_DST
) != receipt_sha:

    raise RuntimeError(
        "Receipt changed before commit."
    )


print(
    "Script exact source bytes:",
    "PASS",
)

print(
    "Receipt bytes            :",
    "PASS",
)


# =============================================================================
# 10. COMMIT
# =============================================================================

banner("STAGE26-NB2 :: COMMIT")


git(
    "commit",
    "-m",
    COMMIT_MSG,
)


new_head = git(
    "rev-parse",
    "HEAD",
)

parent = git(
    "rev-parse",
    "HEAD^",
)

subject = git(
    "show",
    "-s",
    "--format=%s",
    new_head,
)


print(
    "Parent  :",
    parent,
)

print(
    "New HEAD:",
    new_head,
)

print(
    "Subject :",
    subject,
)


if parent != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage26 notebook-source commit has unexpected parent."
    )


if subject != COMMIT_MSG:
    raise RuntimeError(
        "Unexpected commit subject."
    )


# =============================================================================
# 11. PUSH SECURELY
# =============================================================================

banner("STAGE26-NB2 :: PUSH")


from kaggle_secrets import UserSecretsClient


token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)


if not token:
    raise RuntimeError(
        "GITHUB_TOKEN unavailable."
    )


fd, askpass_name = tempfile.mkstemp(
    prefix="stage26_nb2_askpass_",
    suffix=".sh",
)

os.close(
    fd
)

askpass = Path(
    askpass_name
)


try:

    askpass.write_text(
        "#!/bin/sh\n"
        "case \"$1\" in\n"
        "  *Username*) printf '%s\\n' 'x-access-token' ;;\n"
        "  *Password*) printf '%s\\n' \"$TOKEN_VALUE\" ;;\n"
        "  *) printf '%s\\n' '' ;;\n"
        "esac\n",
        encoding="utf-8",
    )


    askpass.chmod(
        askpass.stat().st_mode
        | stat.S_IXUSR
    )


    env = os.environ.copy()

    env[
        "GIT_ASKPASS"
    ] = str(
        askpass
    )

    env[
        "GIT_TERMINAL_PROMPT"
    ] = "0"

    env[
        "TOKEN_VALUE"
    ] = token


    print(
        git(
            "push",
            "origin",
            "main",
            env=env,
        )
    )


finally:

    askpass.unlink(
        missing_ok=True
    )

    token = None


# =============================================================================
# 12. REMOTE HEAD VERIFICATION
# =============================================================================

banner("STAGE26-NB2 :: REMOTE HEAD VERIFICATION")


git(
    "fetch",
    "origin",
    "main",
)


local_head = git(
    "rev-parse",
    "HEAD",
)

remote_head = git(
    "rev-parse",
    "origin/main",
)


print(
    "Local HEAD :",
    local_head,
)

print(
    "origin/main:",
    remote_head,
)


if not (
    local_head
    ==
    remote_head
    ==
    new_head
):
    raise RuntimeError(
        "Local/remote HEAD mismatch after push."
    )


# =============================================================================
# 13. REMOTE BYTE VERIFICATION
# =============================================================================

banner("STAGE26-NB2 :: REMOTE BYTE VERIFICATION")


for rel in expected_files:

    p = subprocess.run(
        [
            "git",
            "show",
            f"origin/main:{rel}",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )


    if p.returncode != 0:
        raise RuntimeError(
            f"Unable to read remote file: {rel}"
        )


    remote_sha = hashlib.sha256(
        p.stdout
    ).hexdigest()


    expected_sha = local_sha[
        rel
    ]


    print(rel)

    print(
        "  expected:",
        expected_sha
    )

    print(
        "  remote  :",
        remote_sha
    )

    print(
        "  PASS    :",
        remote_sha == expected_sha,
    )


    if remote_sha != expected_sha:
        raise RuntimeError(
            f"Remote byte mismatch: {rel}"
        )


# =============================================================================
# 14. FINAL STATE
# =============================================================================

banner("STAGE26-NB2 COMPLETE :: KAGGLE SOURCE DURABLY PRESERVED")


final_status = git(
    "status",
    "--porcelain",
)


print(
    "Stage26 scientific anchor :",
    STAGE26_SCIENTIFIC_ANCHOR,
)

print(
    "Notebook archival HEAD    :",
    new_head,
)

print()
print(
    "Archived script           :",
    repo_rel(
        SCRIPT_DST
    ),
)

print(
    "Script bytes              :",
    f"{SCRIPT_DST.stat().st_size:,}",
)

print(
    "Script SHA256             :",
    script_sha,
)

print()
print(
    "Export receipt            :",
    repo_rel(
        RECEIPT_DST
    ),
)

print(
    "Receipt SHA256            :",
    receipt_sha,
)

print()
print(
    "Source preserved exactly  : YES"
)

print(
    "Source rewritten          : NO"
)

print(
    "Syntax modified           : NO"
)

print(
    "GPU used                  : NO"
)

print(
    "New science               : NO"
)

print()
print(
    "HEAD == origin/main       :",
    local_head
    ==
    remote_head
    ==
    new_head,
)

print(
    "Repository clean          :",
    final_status == "",
)

print()
print(
    "STAGE26 NOTEBOOK SOURCE    : DURABLE / PUSHED"
)


if final_status:
    raise RuntimeError(
        "Repository not clean after notebook-source preservation."
    )


STAGE26-NB2 :: REPOSITORY GATE
Expected parent           : 8767f38719584ec9e8d18e5c1dfae0913e537fed
Local HEAD                : 8767f38719584ec9e8d18e5c1dfae0913e537fed
origin/main               : 8767f38719584ec9e8d18e5c1dfae0913e537fed
Repository clean          : True
Stage26 scientific anchor : 9e8354ecc9cfa72c28aa037e5d2053de422bf7a2

STAGE26-NB2 :: SNAPSHOT KAGGLE VIRTUAL SOURCE
Source path   : /kaggle/working/.virtual_documents/__notebook_source__.ipynb
Source bytes  : 3,807,767
Source SHA256 : e042c75c86b5f43f0da34fdb1bd63f5cb35b744f7003eaf99559324b0738229a
UTF-8 characters: 3,807,410
Source lines     : 210,533
STAGE26 mentions : 5,987

STAGE26-NB2 :: SOURCE FORMAT QUALIFICATION
Starts as JSON object: False
Starts as JSON array : False
First line           : # This Python 3 environment comes with many helpful analytics libraries installed
Standalone compile: False
Compile diagnostic: SyntaxError: unmatched ')' (__notebook_source__.ipynb, line 6331)

Compilation is NOT an archiv